# Append new keys and values to a ragged KV cache

During decoding, every step appends the new token's keys and values to a preallocated cache. Sequences in a batch have different lengths, so each one writes at its own position.

Write `kv_append(k_cache, v_cache, k_new, v_new, seq_lens)`.

## Inputs
- `k_cache`, `v_cache`: contiguous `float32` CUDA tensors `(B, H, S_max, D)`. Positions `>= seq_lens[b]` of sequence `b` hold stale data.
- `k_new`, `v_new`: contiguous `float32` `(B, H, T_new, D)`, with $T_{new} \ge 0$ (1 for a decode step, more for a chunk of prompt).
- `seq_lens`: `int64` `(B,)`, the current length of each sequence. Guaranteed: `seq_lens[b] + T_new <= S_max`.

## What to do
For every `b, h`, `t < T_new`: write `k_new[b, h, t, :]` to `k_cache[b, h, seq_lens[b] + t, :]`, and the same for `v`. Touch nothing else.

## Output
The tuple `(k_cache, v_cache, seq_lens + T_new)`: the two caches updated **in place** (return the same tensors) and the new lengths as `int64`. If $T_{new} = 0$, launch nothing and return the inputs with unchanged lengths.

## Requirements
- Grid `(B * H, T_new)`: one program per (sequence, head, new token) copies one row of `D` elements for both K and V, with `BLOCK_D = max(triton.next_power_of_2(D), 16)` and a mask.
- Compute offsets in `int64`: a real cache has billions of elements.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def kv_append(k_cache, v_cache, k_new, v_new, seq_lens):
    """Write k_new/v_new (B, H, T_new, D) into the caches (B, H, S_max, D) at seq_lens[b] + t. Return (k, v, lens)."""
    raise NotImplementedError


@triton.jit
def _kv_append_kernel(kc_ptr, vc_ptr, kn_ptr, vn_ptr, lens_ptr, H, S_max, T_new, D, BLOCK_D: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[[1.0, 2.0]]], [[[3.0, 4.0]]]], "dtype": "float32"}, {"__nd__": [[[[-1.0, -2.0]]], [[[-3.0, -4.0]]]], "dtype": "float32"}, {"__nd__": [0, 2], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[1.0, 2.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [3.0, 4.0], [0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [[[[-1.0, -2.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]], [[[0.0, 0.0], [0.0, 0.0], [-3.0, -4.0], [0.0, 0.0]]]], "dtype": "float32"}, {"__nd__": [1, 3], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0]], [[9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0]]]], "dtype": "float32"}, {"__nd__": [[[[9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0]], [[9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0]]]], "dtype": "float32"}, {"__nd__": [[[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], [[6.0, 7.0, 8.0], [9.0, 10.0, 11.0]]]], "dtype": "float32"}, {"__nd__": [[[[-0.0, -1.0, -2.0], [-3.0, -4.0, -5.0]], [[-6.0, -7.0, -8.0], [-9.0, -10.0, -11.0]]]], "dtype": "float32"}, {"__nd__": [1], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[9.0, 9.0, 9.0], [0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0]], [[9.0, 9.0, 9.0], [6.0, 7.0, 8.0], [9.0, 10.0, 11.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0]]]], "dtype": "float32"}, {"__nd__": [[[[9.0, 9.0, 9.0], [-0.0, -1.0, -2.0], [-3.0, -4.0, -5.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0]], [[9.0, 9.0, 9.0], [-6.0, -7.0, -8.0], [-9.0, -10.0, -11.0], [9.0, 9.0, 9.0], [9.0, 9.0, 9.0]]]], "dtype": "float32"}, {"__nd__": [3], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[-0.8019314408302307, -1.3243589401245117, -0.24836161732673645, 0.4204452335834503], [1.1360465288162231, 0.10970640182495117, -0.5526472926139832, -0.7847803831100464], [0.7487457990646362, 1.6347830295562744, 0.2727687656879425, -1.2333287000656128], [-0.9582651853561401, 1.600019097328186, 0.20288243889808655, -1.7321348190307617], [-0.08369619399309158, -1.1632260084152222, -0.6292880773544312, -0.4880058169364929], [-0.7133134007453918, 0.5533784627914429, -0.06308597326278687, -0.5894312858581543]], [[0.40963783860206604, 0.8298553228378296, -1.6430233716964722, -0.2567301392555237], [-0.9807473421096802, -0.17315521836280823, -1.2894186973571777, 0.020690394565463066], [-0.03788574039936066, -0.304337739944458, -1.0479265451431274, -0.396190345287323], [-1.0913288593292236, -1.3552087545394897, 0.2247857302427292, -1.1093499660491943], [1.1702960729599, 0.7165876626968384, -1.9978166818618774, 0.2721288800239563], [-1.1017166376113892, 0.03305722028017044, 0.04363199323415756, -1.9884297847747803]]], [[[-0.23342251777648926, -0.25579002499580383, 0.9620005488395691, -1.1814467906951904], [0.738041877746582, -1.0989727973937988, -0.3312909007072449, -0.8404731750488281], [1.4487313032150269, 0.5682131052017212, 2.431732416152954, 0.6419163942337036], [0.8449927568435669, 0.840682864189148, -0.6066115498542786, -0.07002844661474228], [1.3503888845443726, -0.3965507745742798, 0.18879953026771545, -0.021223461255431175], [0.6092165112495422, -0.36490875482559204, -0.1523618847131729, 0.24238142371177673]], [[0.10302314162254333, -0.8649727702140808, 0.8957830667495728, -1.2984812259674072], [-1.201115608215332, -1.2824918031692505, 0.966972291469574, -0.36060836911201477], [-0.9710363745689392, -1.136021375656128, 0.4211311340332031, -1.0548406839370728], [-1.2720781564712524, 0.6139930486679077, -1.1967077255249023, -0.32243815064430237], [-0.006761540193110704, -0.44533535838127136, -0.05409396067261696, 1.338773488998413], [-0.5168941617012024, -1.2593071460723877, -1.8367457389831543, -0.20476610958576202]]]], "dtype": "float32"}, {"__nd__": [[[[-0.35225701332092285, 0.2650909125804901, -0.4642446041107178, -0.4786384701728821], [-0.7213157415390015, -0.5197597742080688, 0.1602267026901245, -0.3803527355194092], [0.10044185072183609, 1.9011871814727783, 0.47911813855171204, -1.576089859008789], [1.7335295677185059, 0.3478103578090668, -0.9414132833480835, 0.9070489406585693], [0.017655611038208008, -0.6152185201644897, -0.6335089206695557, -0.993431806564331], [0.04811898246407509, 1.0688166618347168, -0.3250521123409271, 0.4208241105079651]], [[1.8756459951400757, -1.2145906686782837, 0.257522314786911, -0.3064369261264801], [-1.0592560768127441, -1.0258471965789795, -0.015285143628716469, 0.43386340141296387], [-0.5315898060798645, 0.054390545934438705, 0.7222086787223816, 0.29656124114990234], [0.8686701655387878, 0.46139562129974365, 0.488005667924881, 1.8266340494155884], [0.623171865940094, 0.12947113811969757, 0.7099438309669495, -0.919123649597168], [-0.33770477771759033, 0.7932208776473999, 0.6307867169380188, 1.5483616590499878]]], [[[0.010473028756678104, -1.4623278379440308, 1.9472469091415405, 1.0928927659988403], [-1.0587373971939087, 1.3758236169815063, 0.032402798533439636, -1.8139287233352661], [-0.4203539192676544, -0.5089038610458374, 1.5908482074737549, -0.7920687198638916], [-0.2536173462867737, -0.2767608165740967, -0.37897276878356934, -0.9140740633010864], [0.2190869152545929, 1.0768046379089355, 0.6239776611328125, -0.9274991154670715], [-1.1498157978057861, 0.11897355318069458, -0.7065801620483398, -0.6301656365394592]], [[-1.6796945333480835, 1.9504915475845337, 0.9166191816329956, -0.9739073514938354], [0.908210277557373, 1.342487096786499, -2.3895533084869385, -0.5489447712898254], [-0.38797321915626526, 0.6482544541358948, -0.12146542221307755, -0.23043133318424225], [-0.05837138742208481, 1.8533257246017456, 2.159980535507202, -0.5248264074325562], [-0.9262440204620361, 2.6925530433654785, -0.9797446131706238, -0.5734020471572876], [0.036581508815288544, 0.48324623703956604, 1.0290355682373047, 0.39097723364830017]]]], "dtype": "float32"}, {"__nd__": [[[], []], [[], []]], "dtype": "float32"}, {"__nd__": [[[], []], [[], []]], "dtype": "float32"}, {"__nd__": [3, 1], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[-0.8019314408302307, -1.3243589401245117, -0.24836161732673645, 0.4204452335834503], [1.1360465288162231, 0.10970640182495117, -0.5526472926139832, -0.7847803831100464], [0.7487457990646362, 1.6347830295562744, 0.2727687656879425, -1.2333287000656128], [-0.9582651853561401, 1.600019097328186, 0.20288243889808655, -1.7321348190307617], [-0.08369619399309158, -1.1632260084152222, -0.6292880773544312, -0.4880058169364929], [-0.7133134007453918, 0.5533784627914429, -0.06308597326278687, -0.5894312858581543]], [[0.40963783860206604, 0.8298553228378296, -1.6430233716964722, -0.2567301392555237], [-0.9807473421096802, -0.17315521836280823, -1.2894186973571777, 0.020690394565463066], [-0.03788574039936066, -0.304337739944458, -1.0479265451431274, -0.396190345287323], [-1.0913288593292236, -1.3552087545394897, 0.2247857302427292, -1.1093499660491943], [1.1702960729599, 0.7165876626968384, -1.9978166818618774, 0.2721288800239563], [-1.1017166376113892, 0.03305722028017044, 0.04363199323415756, -1.9884297847747803]]], [[[-0.23342251777648926, -0.25579002499580383, 0.9620005488395691, -1.1814467906951904], [0.738041877746582, -1.0989727973937988, -0.3312909007072449, -0.8404731750488281], [1.4487313032150269, 0.5682131052017212, 2.431732416152954, 0.6419163942337036], [0.8449927568435669, 0.840682864189148, -0.6066115498542786, -0.07002844661474228], [1.3503888845443726, -0.3965507745742798, 0.18879953026771545, -0.021223461255431175], [0.6092165112495422, -0.36490875482559204, -0.1523618847131729, 0.24238142371177673]], [[0.10302314162254333, -0.8649727702140808, 0.8957830667495728, -1.2984812259674072], [-1.201115608215332, -1.2824918031692505, 0.966972291469574, -0.36060836911201477], [-0.9710363745689392, -1.136021375656128, 0.4211311340332031, -1.0548406839370728], [-1.2720781564712524, 0.6139930486679077, -1.1967077255249023, -0.32243815064430237], [-0.006761540193110704, -0.44533535838127136, -0.05409396067261696, 1.338773488998413], [-0.5168941617012024, -1.2593071460723877, -1.8367457389831543, -0.20476610958576202]]]], "dtype": "float32"}, {"__nd__": [[[[-0.35225701332092285, 0.2650909125804901, -0.4642446041107178, -0.4786384701728821], [-0.7213157415390015, -0.5197597742080688, 0.1602267026901245, -0.3803527355194092], [0.10044185072183609, 1.9011871814727783, 0.47911813855171204, -1.576089859008789], [1.7335295677185059, 0.3478103578090668, -0.9414132833480835, 0.9070489406585693], [0.017655611038208008, -0.6152185201644897, -0.6335089206695557, -0.993431806564331], [0.04811898246407509, 1.0688166618347168, -0.3250521123409271, 0.4208241105079651]], [[1.8756459951400757, -1.2145906686782837, 0.257522314786911, -0.3064369261264801], [-1.0592560768127441, -1.0258471965789795, -0.015285143628716469, 0.43386340141296387], [-0.5315898060798645, 0.054390545934438705, 0.7222086787223816, 0.29656124114990234], [0.8686701655387878, 0.46139562129974365, 0.488005667924881, 1.8266340494155884], [0.623171865940094, 0.12947113811969757, 0.7099438309669495, -0.919123649597168], [-0.33770477771759033, 0.7932208776473999, 0.6307867169380188, 1.5483616590499878]]], [[[0.010473028756678104, -1.4623278379440308, 1.9472469091415405, 1.0928927659988403], [-1.0587373971939087, 1.3758236169815063, 0.032402798533439636, -1.8139287233352661], [-0.4203539192676544, -0.5089038610458374, 1.5908482074737549, -0.7920687198638916], [-0.2536173462867737, -0.2767608165740967, -0.37897276878356934, -0.9140740633010864], [0.2190869152545929, 1.0768046379089355, 0.6239776611328125, -0.9274991154670715], [-1.1498157978057861, 0.11897355318069458, -0.7065801620483398, -0.6301656365394592]], [[-1.6796945333480835, 1.9504915475845337, 0.9166191816329956, -0.9739073514938354], [0.908210277557373, 1.342487096786499, -2.3895533084869385, -0.5489447712898254], [-0.38797321915626526, 0.6482544541358948, -0.12146542221307755, -0.23043133318424225], [-0.05837138742208481, 1.8533257246017456, 2.159980535507202, -0.5248264074325562], [-0.9262440204620361, 2.6925530433654785, -0.9797446131706238, -0.5734020471572876], [0.036581508815288544, 0.48324623703956604, 1.0290355682373047, 0.39097723364830017]]]], "dtype": "float32"}, {"__nd__": [3, 1], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[-0.8723830580711365, 0.5063823461532593, 0.24991758167743683, 1.8769190311431885, -0.014999798499047756, -1.336972951889038, -1.0450198650360107, 1.450153112411499, -0.5401312112808228, -2.104465961456299, -0.5806996822357178, 1.5099831216502935e-05, 1.1888307332992554, -1.0144681930541992, 0.6666833162307739, 0.7952991127967834, -0.699388325214386, -0.18758970499038696, 1.7694501876831055, 1.720484733581543, 0.8555219769477844, 0.33194637298583984, 1.1383095979690552, -0.1406772881746292, -0.09509306401014328, -0.8603601455688477, 0.0055633848533034325, -0.08201662451028824, 2.7736315727233887, -0.19284316897392273, 1.2708179950714111, 1.3204972743988037, -0.18647870421409607, 1.169397234916687, -2.1768136024475098, 0.09492718428373337, 0.8572614192962646, -2.3978652954101562, -1.159171223640442, 1.0560736656188965, -0.2570107579231262, -1.098198652267456, -0.373595654964447, -0.5425882935523987, 0.7244473099708557, 0.45388588309288025, -0.2802283465862274, -0.6653426885604858, -0.05387566238641739, 1.3251559734344482, 0.3490261137485504, 0.6389455795288086, -0.15068159997463226, -1.291816234588623, -0.710843563079834, 0.6512365937232971, 0.10321610420942307, -0.9038398861885071, -1.6178704500198364, 0.11068189889192581, -0.3395310044288635, -0.350530207157135, -0.32944995164871216, 0.28390511870384216], [0.29444772005081177, 0.9621285796165466, 0.9284215569496155, 1.3320931196212769, 0.806363046169281, -0.33560803532600403, -0.05426894500851631, 0.49421510100364685, -0.3102922737598419, 0.5494387745857239, 1.1536095142364502, -0.8012459874153137, -1.7703840732574463, 1.193655014038086, 0.017759909853339195, 0.7086698412895203, -1.3323053121566772, 1.7329925298690796, 0.738619863986969, -0.1260741800069809, -0.3519880473613739, 0.11389587819576263, -0.1667313426733017, -0.2786901295185089, -1.1195999383926392, 1.2628093957901, -0.03957768529653549, 0.07463563978672028, 0.09291598945856094, -2.358217239379883, 0.5271414518356323, -0.32915493845939636, -0.2892284691333771, 0.0697878748178482, -0.9724755883216858, 0.8433808088302612, -0.6958646178245544, -0.9036580324172974, -0.8621895909309387, 0.1587526947259903, 0.7457794547080994, -0.5950532555580139, -1.5101866722106934, 1.062490701675415, -1.065083622932434, 1.1948304176330566, -0.05429587513208389, -0.6983498930931091, -0.6400399208068848, 0.461866170167923, 0.8893718123435974, -0.1981702297925949, 0.9469344019889832, 2.0869781970977783, -0.20830833911895752, -1.1494466066360474, -1.0544003248214722, -1.3903353214263916, -0.639665424823761, -0.7237792611122131, -1.3251032829284668, 0.530184268951416, -2.6113390922546387, -1.3207989931106567], [2.015251874923706, 1.0378024578094482, 0.5227916836738586, 0.45725905895233154, 1.770937204360962, 0.5356296300888062, 0.07757708430290222, -1.230160117149353, -0.07366681843996048, -0.7338839769363403, -0.7061752676963806, -0.7142389416694641, -0.044646862894296646, -0.32971081137657166, 0.8711104989051819, -1.326181411743164, -0.1694701910018921, 1.8594975471496582, 0.03889733552932739, 1.1986151933670044, 1.1368634700775146, 1.3166477680206299, 0.7381191849708557, -1.3804460763931274, -0.267387717962265, -0.38033556938171387, -1.5488553047180176, -0.46388161182403564, -1.250079870223999, -1.6655514240264893, 0.21089625358581543, -0.043591272085905075, -0.6878148913383484, -0.9505863785743713, -0.5412734150886536, -0.3940903842449188, -0.40811029076576233, -0.052707746624946594, -0.43554067611694336, 0.38697198033332825, 0.5228708982467651, 0.3578696846961975, -0.6631035208702087, -0.6165061593055725, 0.4348641037940979, -0.197042778134346, -0.04315171018242836, -2.491995334625244, -2.700103998184204, 0.33714818954467773, 0.32955867052078247, 1.3934319019317627, 0.7211630940437317, 0.221738800406456, 0.01636245660483837, -1.4267404079437256, 0.8831759095191956, -0.6035860180854797, 0.5024844408035278, 0.20004165172576904, 0.27209702134132385, 0.8926752209663391, -0.48986610770225525, -0.38723519444465637], [-0.855861485004425, -0.3771520256996155, 1.665940761566162, -0.36331865191459656, 1.1856162548065186, 0.8260114192962646, 0.7292817831039429, 0.8030151128768921, -1.0142765045166016, 0.3734476864337921, 1.5884120464324951, -0.6957959532737732, -0.04710708558559418, -0.6891591548919678, 0.3170115053653717, -1.0785300731658936, 1.6096798181533813, -0.37683477997779846, -1.5426615476608276, 1.7907909154891968, 0.9705228805541992, 0.5185114741325378, 0.5325875282287598, -0.3577980101108551, -0.22329559922218323, 0.7240192294120789, 0.9176249504089355, 0.932612419128418, 0.21255098283290863, -0.46638810634613037, -0.192087322473526, 1.1095997095108032, 1.4399181604385376, -0.34131237864494324, -0.47730401158332825, 1.2475496530532837, 0.7236490249633789, 0.6972581744194031, -1.3123337030410767, -0.3737574517726898, -0.5149955153465271, 0.207904651761055, -1.6849420070648193, 1.2717528343200684, 0.30013179779052734, 0.4348370134830475, -1.1841403245925903, -0.02642037160694599, 1.3346502780914307, -1.5696955919265747, 0.30576804280281067, 0.6988049149513245, -0.6919471025466919, 0.8147412538528442, -0.33006903529167175, 1.229766607284546, 0.5275117754936218, 0.22436589002609253, 2.008915901184082, 0.8318964838981628, 1.3086357116699219, -1.1895196437835693, -0.6466070413589478, 0.8365690112113953], [0.9056118726730347, -0.055740028619766235, 0.779726505279541, -1.0583537817001343, 0.2138352245092392, -1.147283911705017, 0.3405705392360687, -0.4861719012260437, -0.6409975290298462, 0.056299030780792236, 1.817460060119629, -1.0117617845535278, -0.6017506122589111, 1.0676641464233398, -0.42356666922569275, 1.7692642211914062, 0.22252970933914185, 0.09063191711902618, 0.3568841516971588, 1.7411795854568481, 0.655361533164978, 0.5167121291160583, 0.2518577575683594, -1.562418818473816, -0.8852720260620117, 1.4833440780639648, 0.8218783140182495, 0.5060648322105408, 0.6370627284049988, -0.5435930490493774, 0.33796426653862, -0.9007946252822876, 0.3913623094558716, -0.7141351103782654, -0.21381983160972595, 0.4534148573875427, -0.32678771018981934, 0.647581934928894, 0.4560382664203644, 0.35945814847946167, 1.324411392211914, -1.4738847017288208, -1.5962822437286377, 0.6162418127059937, 0.9407941102981567, 2.0116403102874756, 0.7562568187713623, 0.8302149772644043, -0.8830800652503967, 0.7526217699050903, 0.6738389134407043, -1.0558651685714722, -0.8963174819946289, -0.11103120446205139, 0.2882537543773651, -2.9447855949401855, 0.07030551135540009, 0.23085449635982513, -0.17683635652065277, 0.2668249011039734, 0.4559062421321869, -1.0116380453109741, 0.41618016362190247, -0.2513071894645691], [-0.945878803730011, 0.20914079248905182, 1.4876441955566406, -1.4601155519485474, -1.6531099081039429, -1.6531661748886108, 1.9773117303848267, -0.2515837550163269, -0.0594867542386055, -0.7865757346153259, -0.9404478073120117, -2.026860237121582, 0.1704062968492508, 0.2595598101615906, -1.9765340089797974, -1.0431492328643799, -0.7441137433052063, -0.02600245364010334, 0.24412916600704193, 1.1257929801940918, 1.364456057548523, 1.0684545040130615, 1.3415285348892212, -1.1664166450500488, -0.5544917583465576, 1.1436097621917725, 0.28866109251976013, 0.018721718341112137, -1.9789881706237793, -1.8592429161071777, 0.7308894395828247, 0.34157639741897583, -0.2960576117038727, -0.41297173500061035, -1.726102590560913, 0.3712151348590851, 1.0344208478927612, 1.642171025276184, 0.04913794994354248, -0.20419667661190033, -1.0959970951080322, 1.2232437133789062, 1.7068555355072021, -0.6131700277328491, 0.29075875878334045, 0.09067010879516602, -0.57382732629776, -2.4974496364593506, -0.36599910259246826, 0.607444167137146, -1.443302035331726, -0.838881254196167, -3.2835605144500732, 1.2845276594161987, 0.5002763867378235, 0.04547853022813797, -0.2704733908176422, -0.637283205986023, -0.07880625128746033, -0.31031492352485657, -0.24835671484470367, -0.033811114728450775, 0.12565097212791443, 1.5489784479141235], [-1.449432373046875, 0.26196861267089844, -0.6245608329772949, -1.1394811868667603, -2.368170738220215, 0.6068921685218811, -0.9903792142868042, -0.7596859931945801, 0.45987868309020996, -1.093472957611084, 0.016045130789279938, -0.08488040417432785, 0.7918434143066406, 1.0006409883499146, -0.8768518567085266, 0.907177746295929, -0.1067456603050232, 0.1127975657582283, -0.2225136160850525, 0.7655396461486816, -1.2222003936767578, 0.2500842809677124, 1.1657798290252686, -0.2936699092388153, 2.0177724361419678, -0.745905876159668, 0.2036961317062378, -0.49260851740837097, -0.6287503242492676, -1.1257710456848145, -0.5500664710998535, -0.06223707273602486, -0.8130971789360046, 0.8723957538604736, 1.510326623916626, -1.5388962030410767, 0.8248433470726013, 0.17806008458137512, 0.2333207130432129, 2.4504992961883545, 1.1943246126174927, 0.3243098556995392, 0.17311415076255798, -2.165717124938965, 0.7723909020423889, -0.7486869692802429, 0.17573262751102448, 0.09763062745332718, 0.9603515267372131, 2.0305027961730957, 0.9836022853851318, 1.2729486227035522, -0.23454320430755615, 0.4557735025882721, 0.769248902797699, -1.43962824344635, 2.0415430068969727, -0.7148682475090027, -1.277543306350708, 1.6340441703796387, 2.118438720703125, -0.7031057476997375, -0.6133099794387817, 3.111912965774536], [0.1126171126961708, -1.984058141708374, -0.6122687458992004, 2.0021188259124756, 0.6591240763664246, 0.7520990967750549, 1.2245573997497559, 0.6757786870002747, -0.3332044780254364, -1.5943117141723633, 0.44637084007263184, -1.1160348653793335, -0.05623851716518402, 1.758971095085144, 0.5406503081321716, 0.05949153006076813, 0.06568005681037903, 0.730792760848999, 0.9212337732315063, -1.3168138265609741, 0.1854793280363083, 1.7156471014022827, 0.7543686628341675, -0.7651258707046509, -0.6796277165412903, -2.061922550201416, -0.7933788299560547, 1.6069653034210205, -0.0695239007472992, -0.855010986328125, -0.5761204957962036, -0.11074551194906235, -0.10654617100954056, -0.35397371649742126, -0.7690944671630859, 0.6365700960159302, 1.5385345220565796, -0.5679569840431213, -0.5412718653678894, 1.189353108406067, -1.4637436866760254, -0.5913947224617004, -0.3780970871448517, 0.16778159141540527, -0.27493882179260254, -0.8866033554077148, 0.31050634384155273, -1.1877104043960571, -1.742513656616211, -2.0282576084136963, 0.7486985325813293, -1.7368555068969727, 0.4470572769641876, 0.6707091331481934, 0.8068993091583252, -0.17468272149562836, -0.5482001304626465, 0.0856713354587555, -0.5937349200248718, -1.286202311515808, 1.2554398775100708, -0.8944374322891235, 1.7958890199661255, 0.5147645473480225], [-0.45455676317214966, 0.43764764070510864, 1.230149507522583, -1.3784692287445068, -0.926772952079773, -1.0492675304412842, 0.16503140330314636, 0.4042508602142334, 0.028968343511223793, -0.7282098531723022, -0.5284513235092163, 1.187043309211731, 1.1169426441192627, 0.13135528564453125, -1.310608983039856, 0.6281268000602722, 0.2876591682434082, 0.25273048877716064, 1.7300316095352173, 0.8205397129058838, -0.974485456943512, -0.9647605419158936, 1.3803138732910156, 0.20571652054786682, 0.8870580196380615, -0.10639959573745728, -0.9494908452033997, 0.32420140504837036, 1.6189154386520386, 0.39913585782051086, -1.065564751625061, -0.24585874378681183, -0.42841270565986633, 0.2473440170288086, 1.0980541706085205, 2.0542898178100586, 0.3144912123680115, -0.3364866077899933, 0.1715981662273407, 0.11086616665124893, -0.32979634404182434, 0.1186041608452797, 0.09529157727956772, -1.0805187225341797, 0.6438245177268982, -0.2667025625705719, -0.8877440690994263, 0.5859832167625427, -0.39159297943115234, 1.8382551670074463, 0.5709066987037659, 0.1686367392539978, -0.665776252746582, -0.31839120388031006, 0.9355282783508301, -0.6024860739707947, 0.8436412811279297, -2.0179641246795654, 0.28045257925987244, 0.8953021764755249, 0.41629400849342346, 0.07274018973112106, 1.086340069770813, -0.7792912125587463], [-0.9079190492630005, 0.1802140772342682, 1.2776236534118652, 0.5065013766288757, -0.11938237398862839, 0.37507161498069763, 0.5171626806259155, -1.1704579591751099, -2.7371013164520264, -0.950610339641571, -1.4196820259094238, -0.5050193071365356, -1.1275153160095215, -0.7912658452987671, 0.5087776780128479, 0.17351014912128448, 0.05337090045213699, 1.7401034832000732, -0.3785295784473419, -0.2968817353248596, 1.3511508703231812, -1.025349497795105, -1.5425773859024048, 1.397329568862915, 0.42847737669944763, 0.6939471364021301, 0.7324030995368958, 1.000512719154358, -0.37342125177383423, 0.024766210466623306, -1.0105915069580078, 0.24289265275001526, 0.3374541997909546, -0.3463873863220215, 0.7765704989433289, 3.064624071121216, -1.0395714044570923, 0.9449557662010193, 1.060904860496521, 1.186570167541504, -0.6814988851547241, 1.3214032649993896, 0.7814787030220032, 0.004755706060677767, -0.21026098728179932, 0.3801894783973694, 1.071524977684021, 1.024174451828003, 2.4328603744506836, -0.2064686417579651, 0.7699827551841736, -0.2774447500705719, 0.9362072348594666, 0.28631535172462463, -0.44236698746681213, -1.170732021331787, -0.6131279468536377, 0.3971872627735138, 1.6269680261611938, 0.9308051466941833, -1.0413693189620972, -0.33361268043518066, 0.1840648353099823, 0.01132616400718689], [0.19017614424228668, 0.8913050889968872, 0.31386786699295044, -0.601158618927002, -1.8446869850158691, -0.2656921446323395, -1.6127432584762573, -0.10446330159902573, 0.363433837890625, -1.2998757362365723, -0.11584888398647308, -0.3271665871143341, 1.906078815460205, 0.44876807928085327, -1.9007844924926758, 0.42046672105789185, 0.3862980902194977, -0.2349327951669693, -0.4137638509273529, 0.5544675588607788, 0.6138501167297363, 0.38203006982803345, -1.0521268844604492, -0.3142268657684326, -1.6254591941833496, -1.566889762878418, -0.6347933411598206, -1.1325814723968506, -0.4956102967262268, 2.3927721977233887, 0.4025426506996155, -0.9809014201164246, -2.2614598274230957, -0.9428343772888184, -1.5433629751205444, -2.3115298748016357, 0.6050446629524231, -0.6025458574295044, 0.3164512813091278, 2.966238260269165, 0.14657582342624664, -0.0074188197031617165, 0.22416597604751587, -0.391497403383255, -0.9520021677017212, -0.3374938666820526, 1.0182347297668457, -0.3933661878108978, -1.492048978805542, -1.368557095527649, -1.1497435569763184, -0.1072094589471817, 0.3657081127166748, 0.36833882331848145, 1.0810695886611938, -0.30665305256843567, 0.7394214272499084, 0.23684793710708618, 0.6244451403617859, -1.9157264232635498, 1.1791285276412964, -1.6059455871582031, -1.1812087297439575, 0.728539764881134], [-0.43165335059165955, 1.728952169418335, 0.12705281376838684, -1.0572758913040161, 0.128393292427063, -0.42599353194236755, 1.5300532579421997, 1.6091804504394531, 0.7201277017593384, -0.42249882221221924, 0.06931523978710175, 1.0352593660354614, -0.8136550784111023, -0.8067298531532288, 0.8386394381523132, 0.7259921431541443, -0.2420131117105484, -0.2782663106918335, -1.9611250162124634, 1.173648715019226, -0.1261032074689865, -1.4362707138061523, -1.1350212097167969, -0.9003010988235474, 1.6969619989395142, 0.6850574612617493, -1.341442346572876, 1.8512842655181885, 1.3351792097091675, -1.0612409114837646, -2.66099214553833, 0.9614986181259155, -0.6027781367301941, -0.13628479838371277, -0.524908185005188, 1.2592880725860596, 1.4789767265319824, 0.810117781162262, -0.15063685178756714, 0.26184359192848206, 0.6211815476417542, -1.5746363401412964, 1.2433815002441406, -0.31869974732398987, 0.26307061314582825, 0.14000190794467926, 0.9083163738250732, 1.0714818239212036, 2.2131998538970947, -0.34747856855392456, -0.8965520858764648, 0.14481224119663239, -0.6418266892433167, 0.8751027584075928, -0.7909221649169922, 0.15036310255527496, 0.9588245153427124, 1.034327507019043, 0.22643126547336578, 1.4341362714767456, 1.3108830451965332, -1.6749932765960693, 0.3853166103363037, 0.20764310657978058], [-0.2698063850402832, -0.607272744178772, 0.07507789134979248, 0.41873955726623535, 0.3086544871330261, 1.3998866081237793, 1.0281139612197876, -0.5532593727111816, -0.7224798202514648, 0.8842297196388245, -0.6599516868591309, 0.20378567278385162, 0.5136725306510925, 1.0678472518920898, 0.5548705458641052, 1.483630657196045, -0.41991153359413147, 0.7100406289100647, -0.6179177165031433, 0.22685877978801727, 0.16443012654781342, -0.8879100680351257, -2.5632784366607666, 0.9970925450325012, 0.4042292833328247, 0.18224069476127625, -0.6591445803642273, -0.29145991802215576, 0.4969172775745392, 0.06155714392662048, 1.200844168663025, 0.1582016795873642, -0.7803479433059692, 0.6506232023239136, -0.593580424785614, 0.2554992139339447, -1.7722288370132446, -0.31255680322647095, 0.5930131077766418, -0.6167822480201721, 0.005562063306570053, 0.6214967966079712, 0.09812398999929428, 1.8157238960266113, -0.22971957921981812, -0.15062332153320312, -0.13217906653881073, -1.9484788179397583, -1.505789041519165, 0.5606258511543274, -0.30178529024124146, 1.41615629196167, -0.05088788643479347, -0.1429254710674286, 1.0557904243469238, 0.3035111725330353, 1.6309014558792114, -1.0298982858657837, 0.46728771924972534, 0.7073956727981567, 0.5844882130622864, 1.1814675331115723, 1.0607037544250488, 0.0700731948018074], [-0.5216994285583496, 1.0194790363311768, 0.0929674580693245, 1.0061924457550049, 1.4043821096420288, 0.25849786400794983, -0.8391388058662415, -0.188700869679451, 0.8577681183815002, -0.16371336579322815, 1.2553577423095703, 0.12203249335289001, 0.21839630603790283, -0.5367452502250671, -0.008135730400681496, 1.1862748861312866, -0.25078389048576355, -0.24314925074577332, -0.6909047365188599, 0.8662309646606445, -1.438070297241211, -0.5731570720672607, 0.9874779582023621, 1.4973156452178955, -0.1496429741382599, 1.033983588218689, -1.914989709854126, 0.8238314390182495, 0.35727834701538086, -1.1592878103256226, -1.716760277748108, -0.44351816177368164, 0.32016366720199585, 0.4285535216331482, 0.17032885551452637, 0.8300293684005737, 0.7591947317123413, -1.325154423713684, -0.3256736993789673, -0.4546818733215332, 1.082566738128662, -0.4368080496788025, 1.0607962608337402, 0.14688044786453247, 2.4004673957824707, 0.005745717324316502, 0.059517305344343185, -0.056776802986860275, 0.20182116329669952, 2.482877731323242, -0.4495837688446045, -0.8182248473167419, -0.14498426020145416, 0.7236251831054688, -0.9369282126426697, 0.8460574746131897, 0.8127056360244751, -0.1168634369969368, -0.2049449384212494, -0.7416505217552185, -0.22885537147521973, -0.7548636794090271, 1.2864465713500977, 1.6693135499954224], [-0.24161919951438904, 0.00040231773164123297, -1.8577793836593628, -0.48929232358932495, 0.3772006034851074, 0.3102934658527374, 0.15976007282733917, 0.880998969078064, 0.1265142410993576, -0.35277992486953735, 1.6342995166778564, -1.8823978900909424, -1.1231029033660889, -0.7060480117797852, -0.6988998055458069, -0.7454692721366882, -0.19863399863243103, -0.6876280903816223, -0.03457644209265709, 0.6096417307853699, 0.046772900968790054, 0.14978359639644623, -0.9177576303482056, -1.1184498071670532, -0.2779383361339569, 0.936154842376709, 0.3869842290878296, 1.095211386680603, 1.0011459589004517, 1.130170226097107, -0.793280839920044, 1.2445451021194458, 0.765385091304779, 2.0772807598114014, 0.597710132598877, 0.3354604244232178, -0.1174328625202179, 0.1754874438047409, -1.6325438022613525, -0.23683799803256989, -0.026065850630402565, 0.9965258836746216, -0.6405432224273682, 0.14371009171009064, 1.8275507688522339, -0.5140311121940613, 0.8498762845993042, -0.5180673003196716, -0.08717534691095352, 0.46998757123947144, 0.8892123699188232, -0.4767798185348511, -1.0020980834960938, 1.5188612937927246, 0.19799013435840607, 0.8002609014511108, 0.9185786843299866, 0.7643093466758728, -0.24615730345249176, -0.12190839648246765, -1.5274707078933716, 0.1913345605134964, -0.4519769847393036, -0.009605282917618752], [1.86375093460083, 0.6937499642372131, 0.06547141075134277, -1.3327445983886719, 0.7337110638618469, 1.5181533098220825, -0.8004674315452576, 0.43257465958595276, -0.49807146191596985, -0.1366279423236847, -0.3479275703430176, -1.5764728784561157, -0.49307113885879517, -1.5852549076080322, 1.3782265186309814, 1.0388894081115723, 0.05506785586476326, 0.2867772877216339, 0.0687292069196701, 0.20761720836162567, 0.20490533113479614, -0.22359073162078857, -0.15187609195709229, -1.3104325532913208, -0.9730347990989685, -0.5838696360588074, 0.21642760932445526, -0.3533463776111603, 1.885884165763855, -0.173744335770607, -0.337854266166687, 0.4379376769065857, 1.0245460271835327, 1.833386778831482, 0.9708805084228516, -0.5633888840675354, -0.15201693773269653, 1.0467700958251953, 0.26900050044059753, 0.8014563322067261, -0.05585052818059921, 1.5882296562194824, 1.437668800354004, 1.1231521368026733, 0.1743306666612625, -1.5570294857025146, -1.299076795578003, 0.3423963189125061, 0.6638202667236328, -2.8262147903442383, -0.4391079843044281, 1.1521999835968018, 0.9599878787994385, -0.4598434567451477, 0.6500676274299622, 0.057154640555381775, -0.185583233833313, 1.1234819889068604, 1.1430305242538452, -0.4834875762462616, -0.8938437700271606, -1.2477530241012573, -1.35883367061615, 1.3349007368087769]], [[1.8648263216018677, 0.24048985540866852, 1.062922477722168, -0.6329414248466492, -0.03174551576375961, 0.9313143491744995, 0.34497955441474915, 0.0025368696078658104, -1.5862555503845215, -1.3807862997055054, -0.36637210845947266, -0.5022204518318176, -0.03396618366241455, -0.584236741065979, -0.17405374348163605, 0.0976613238453865, 0.5410217642784119, 0.10228528082370758, -0.49703362584114075, -0.23362646996974945, 0.9192822575569153, -0.16407977044582367, -0.9060814380645752, -0.9460440874099731, 0.31115853786468506, 0.5796923041343689, 1.2263473272323608, -0.27965250611305237, 1.3735370635986328, -0.3059794306755066, -0.1946568638086319, -0.5217711329460144, 0.5411524176597595, 1.05014967918396, 0.004995408002287149, -0.45856037735939026, -1.4729292392730713, 0.1721293330192566, -0.3063606917858124, -0.1690288484096527, -0.5875093340873718, -0.825576901435852, -0.3926343321800232, 0.45955032110214233, 0.9546487927436829, 0.01128454227000475, 0.3360567092895508, -0.17004065215587616, -0.0998295247554779, -1.3070999383926392, 0.8779817819595337, 0.599260687828064, 1.4344311952590942, -0.5745570063591003, 0.1446526050567627, -0.8759451508522034, 0.2539585530757904, -0.09897676855325699, 0.6767484545707703, -0.12040253728628159, 0.6449835300445557, -0.4546484053134918, 0.5981537103652954, -0.2716369926929474], [-0.4019356966018677, 0.1071111410856247, 0.3789922595024109, -1.3443832397460938, -1.4471772909164429, 0.20568685233592987, -0.03210219740867615, -1.9469980001449585, -1.6273365020751953, -0.7208101153373718, 0.33340221643447876, 0.22141604125499725, 0.999830424785614, 1.159284234046936, 2.3200325965881348, -1.3723598718643188, -0.013785125687718391, 1.0572185516357422, -1.1914669275283813, -1.340640902519226, 0.21681663393974304, 1.0876883268356323, 1.2406551837921143, -1.6978176832199097, -1.151814579963684, -0.06735824048519135, -0.48677846789360046, 0.3394567668437958, 0.15675535798072815, 1.7359110116958618, -0.033442456275224686, 0.2049727439880371, 0.10196290910243988, 1.1016439199447632, -1.0495768785476685, 0.4038074314594269, -0.7196693420410156, -1.3453353643417358, 1.1762868165969849, 0.5426834225654602, 1.3904682397842407, 0.2827655076980591, 0.6671103239059448, 0.16937315464019775, 0.5917483568191528, -0.3609093725681305, 1.701602578163147, -0.8201969861984253, 1.7057604789733887, -0.1485859602689743, 0.8921769261360168, 0.18080472946166992, 0.4822855293750763, 1.4990304708480835, 0.18873842060565948, -0.8698955178260803, 0.2848009467124939, 0.684927225112915, -2.343674421310425, 1.005519986152649, 1.1186068058013916, -1.675768494606018, -0.09786126762628555, 0.8562725186347961], [-1.0792711973190308, -0.3587975800037384, -0.27950209379196167, 0.6004593372344971, 1.143613576889038, 0.02294977754354477, 0.1153683140873909, 0.9619436860084534, 0.5853611826896667, -0.05394911393523216, 1.3585437536239624, -1.3352926969528198, 0.2020653635263443, -0.3604046702384949, -0.44249820709228516, 0.24635453522205353, 0.08954466134309769, 2.466902732849121, 0.6843624114990234, 0.3969602584838867, 1.674640417098999, 1.0493732690811157, -1.7812647819519043, 0.17044976353645325, 0.781863272190094, -0.5458417534828186, 0.33988142013549805, -0.4015931189060211, 0.4161466658115387, -1.5575698614120483, -1.874250888824463, -0.4973315894603729, 0.26492151618003845, 1.3531397581100464, -0.232381671667099, -0.6981062293052673, 0.9420226812362671, 0.07280247658491135, 1.8154182434082031, -0.0507391132414341, -1.8845176696777344, -0.7511929869651794, -0.9051104784011841, -1.3399128913879395, -0.7324886918067932, -0.6487211585044861, -0.25051265954971313, -0.5116637349128723, 0.4315355718135834, 0.4358985722064972, 0.20217981934547424, 0.22221021354198456, -1.8342421054840088, -1.6215094327926636, -0.8437504768371582, 0.27673444151878357, -1.4215724468231201, -0.3913985788822174, 0.2697241008281708, 1.9867075681686401, 0.28176867961883545, -0.5059260725975037, 1.1024010181427002, 0.22433927655220032], [-0.27569183707237244, 1.1198322772979736, -0.8906885981559753, 0.2054339051246643, 0.09374996274709702, 0.9072456955909729, -1.1242132186889648, -1.2728910446166992, -0.37835124135017395, -0.2663916349411011, 0.04562491178512573, -0.3687709867954254, 0.42910754680633545, 0.6845958828926086, 0.22120583057403564, -1.0464434623718262, 1.1411778926849365, -1.5240790843963623, -0.20895320177078247, 0.8774041533470154, -0.6181960701942444, 1.2746925354003906, 1.8907825946807861, 1.5551873445510864, 0.42311325669288635, 0.9650139212608337, -0.5183897614479065, 0.3883185386657715, 0.4431073069572449, 0.0834994912147522, 0.14439734816551208, -0.21514803171157837, -1.957853078842163, 0.38814401626586914, 0.3022112250328064, 2.0293054580688477, 0.6977156400680542, 0.058108072727918625, 0.8594762086868286, 0.6098951697349548, -0.025558190420269966, -0.3873504102230072, -1.4679991006851196, 0.11340352892875671, 1.6074281930923462, -1.8636541366577148, -0.9536433219909668, 0.7432786226272583, 0.18359945714473724, 0.4093780219554901, -1.857773780822754, -0.5945903658866882, 0.4945397675037384, 0.5778988003730774, 0.28581249713897705, -0.8860844969749451, 0.5290326476097107, -1.1968166828155518, -0.9440393447875977, -1.1677287817001343, 1.425971269607544, -0.41940295696258545, 0.046004146337509155, -2.1984241008758545], [1.0813047885894775, 0.45218610763549805, -1.6580231189727783, -1.609822154045105, 1.6429228782653809, -0.9511963725090027, 1.8240394592285156, -0.4779016375541687, -2.4858362674713135, 1.09334135055542, 0.4894861876964569, 0.6681703925132751, -0.5189090371131897, -0.22533050179481506, 0.9421473741531372, -0.8329755663871765, -1.975229024887085, 0.8923466801643372, -0.6322474479675293, 1.2579083442687988, -0.49562975764274597, -0.44075384736061096, -0.26612457633018494, -2.3663153648376465, 1.1586400270462036, -0.6432483196258545, -0.21072284877300262, 1.4466402530670166, 0.09996433556079865, 0.8790737986564636, 0.3202430307865143, 0.7567090392112732, 0.2622949481010437, 1.1387934684753418, -0.22413885593414307, 0.4465140104293823, -0.7214215993881226, 1.1713738441467285, -2.1937522888183594, 0.17275454103946686, 0.3479059636592865, 0.32904183864593506, -0.01979093626141548, 1.3260551691055298, 2.101428985595703, -0.6073958277702332, -1.9200876951217651, -1.0991789102554321, -0.8243827819824219, -0.148102268576622, -0.06763698905706406, 1.0654654502868652, -0.43930330872535706, -0.04962938278913498, 0.09713341295719147, -0.8906799554824829, -0.3303353190422058, 2.742591381072998, -0.10513272136449814, -0.975318193435669, -2.1901962757110596, 1.0241239070892334, -0.335629940032959, -0.6474533081054688], [2.4308433532714844, 1.4182775020599365, -1.7253302335739136, -0.3614228069782257, 0.9838337302207947, 0.8207517862319946, -1.127099871635437, 1.762094259262085, 1.835807204246521, -1.122765302658081, -1.7803300619125366, -1.405688762664795, 0.6507818102836609, 1.0621857643127441, 0.16741704940795898, -0.6129648089408875, -0.1111510768532753, 0.5452548861503601, -1.2326768636703491, -1.7168357372283936, -0.105614073574543, 1.2133082151412964, 0.16984093189239502, 0.17263193428516388, -0.6962419748306274, 0.8977471590042114, -1.7302011251449585, -0.0884302407503128, -0.6173286437988281, -0.17867448925971985, -0.9521907567977905, 0.7226251363754272, -0.10454924404621124, -0.6244663000106812, 0.48045769333839417, -0.08159482479095459, -0.5501055717468262, -0.6215181350708008, 0.7462332844734192, -0.9719409346580505, -0.25569018721580505, -0.3024829626083374, 0.2788255214691162, -0.2284586876630783, 2.1212375164031982, -1.7823542356491089, 0.5563126802444458, -0.06899502873420715, 1.0719588994979858, -1.661516785621643, 1.1792610883712769, -1.4204144477844238, 0.6526589393615723, 0.4729805290699005, -0.007830378599464893, -0.7760050892829895, -0.8554891347885132, 0.05861421674489975, -0.7905116081237793, -1.9903665781021118, -1.847069501876831, 0.7606856226921082, 0.785500705242157, 0.47872257232666016], [0.012259075418114662, 0.3462595045566559, 0.46212875843048096, 0.21987619996070862, -0.8225317597389221, -0.7628558874130249, -1.1806305646896362, 0.12368922680616379, -1.0183985233306885, -0.18521061539649963, 1.2895960807800293, -0.4007664918899536, 0.9477011561393738, 1.5366467237472534, -1.8462074995040894, -1.4550038576126099, 0.4535783529281616, -0.727559506893158, 0.5174764394760132, 0.6908771395683289, -0.5290382504463196, 0.8123423457145691, 1.7308964729309082, 0.9444819688796997, -0.8852678537368774, -0.15462060272693634, -0.5446020364761353, 1.5613423585891724, -0.19860419631004333, -0.6944221258163452, 0.04367179423570633, 0.5124685764312744, -0.37633129954338074, 0.10705791413784027, 0.0024917887058109045, 0.7415816783905029, 0.9778999090194702, 0.27968519926071167, -0.4603579342365265, 0.4139566421508789, -0.8825797438621521, 1.0847476720809937, 0.36036601662635803, 0.2574433386325836, -2.4102981090545654, 2.384186029434204, 1.4711592197418213, 1.288474440574646, 0.147059366106987, 0.6938448548316956, 2.8812804222106934, -0.6559854745864868, 0.7759380340576172, 0.6342946887016296, 0.17647245526313782, 0.3914646506309509, -0.9390909075737, 0.6915391087532043, -1.263176679611206, 1.8821666240692139, 0.8521352410316467, -0.7967486381530762, -2.0363402366638184, 0.16343924403190613], [0.5680439472198486, -0.9327842593193054, 1.741498589515686, 0.3390537202358246, -0.39467480778694153, -1.3252567052841187, -0.04735438898205757, -0.12439973652362823, 0.7260642051696777, 1.125786304473877, 0.10802280902862549, 0.5389137864112854, 1.5030630826950073, 0.6967734694480896, -0.17905555665493011, 0.6983010768890381, -0.44238340854644775, 0.5312957167625427, 0.13035549223423004, -1.0187915563583374, 0.6830295920372009, 0.16322802007198334, 0.8106619119644165, -1.24861478805542, 1.18617582321167, 1.2288119792938232, -0.2741584777832031, -0.022449886426329613, 0.12253140658140182, 0.052425842732191086, 0.9073825478553772, -0.6430671811103821, -1.0808987617492676, -0.1760164350271225, 1.962971568107605, -0.3276088237762451, -0.9405286908149719, -0.8623668551445007, 0.22092878818511963, 1.228193759918213, -0.6862431168556213, 0.22234377264976501, 0.2604827582836151, 0.5109481811523438, 0.18895459175109863, 1.0941689014434814, 0.7297549843788147, -0.5733395218849182, 2.4672462940216064, -1.3218557834625244, -1.2772600650787354, 1.0755003690719604, 0.22601525485515594, -0.20893940329551697, 0.23949386179447174, 0.724484384059906, -0.38553953170776367, -0.5983303189277649, -1.0125545263290405, -0.8265764117240906, -0.7659786343574524, 1.1788495779037476, 0.3116062581539154, 0.5756569504737854], [-0.5320655107498169, 1.0790150165557861, -1.5308595895767212, 0.5036996603012085, 0.7234464287757874, -0.5512312650680542, -0.9989850521087646, 0.5466575026512146, -0.684355616569519, -0.8702754378318787, 0.0020186363253742456, -1.0957164764404297, -0.11396929621696472, 2.049099922180176, -0.41492578387260437, 0.9298009872436523, -1.0312168598175049, 0.10533149540424347, -0.2494734525680542, -0.14563637971878052, 0.7782558798789978, 2.1698338985443115, -0.1063050627708435, 1.3247079849243164, 0.3704155683517456, 0.7564952969551086, -0.44869694113731384, -0.6872261762619019, -0.1815694272518158, -0.7536616921424866, 0.009123672731220722, -1.0375372171401978, 0.023145683109760284, 0.19463421404361725, 1.8339016437530518, 1.7419975996017456, 0.5283176302909851, -0.4836968779563904, 0.6267964839935303, 0.8358477354049683, -0.17668025195598602, -0.8293713331222534, -1.771276593208313, -0.01900007575750351, 1.9416769742965698, -0.1866835653781891, 0.4056864082813263, 0.3856768012046814, 0.5051284432411194, 0.7945407032966614, -1.9836710691452026, -0.12875913083553314, 0.3988375961780548, -0.06705044955015182, 1.2387086153030396, -0.3480604887008667, 0.9469493627548218, 1.7920414209365845, -0.2254084050655365, 0.057759691029787064, -0.2623107433319092, 0.7974582314491272, -2.1079933643341064, -1.4305156469345093], [-0.2600786089897156, 0.9074793457984924, 0.6277872323989868, -0.7818407416343689, 1.6447612047195435, 0.1508508175611496, -0.585057258605957, 0.5723842978477478, 1.3474761247634888, -0.024584978818893433, -1.475991129875183, -0.5078163146972656, 0.13803061842918396, 0.9861943125724792, -0.958077609539032, 0.30879470705986023, -1.1394537687301636, 0.12330487370491028, -1.2868380546569824, -0.41652217507362366, -1.4775577783584595, -0.9329387545585632, -1.544848918914795, -1.9235820770263672, 1.2739397287368774, 1.228738784790039, 0.5300590991973877, -0.30698293447494507, 1.5223150253295898, -0.229792520403862, 0.4800225496292114, 1.1913615465164185, 0.279014527797699, 1.5368289947509766, -0.08661879599094391, -0.09483730792999268, 0.19175629317760468, -1.0959360599517822, -1.0146629810333252, 0.8395438194274902, -1.6665738821029663, 1.1950830221176147, 0.33906376361846924, 1.2519441843032837, 0.8313003182411194, 0.668026864528656, 1.8131299018859863, -0.0066193523816764355, -0.03390422835946083, 1.1996455192565918, -0.5904881954193115, 1.2064158916473389, 0.7611380219459534, -0.43198567628860474, -1.2055048942565918, -1.955995798110962, 1.9460760354995728, -1.4253811836242676, -1.141318440437317, 0.9324416518211365, 0.5389015078544617, 0.2745162844657898, 0.14609888195991516, -0.3657054603099823], [-0.8646954298019409, -0.6852208971977234, 0.6529102921485901, 1.1334609985351562, 1.2387707233428955, 0.014396779239177704, -2.2000515460968018, 0.5596252679824829, -0.0993688553571701, -2.6409964561462402, 1.4376649856567383, 0.2722074091434479, -0.0033549123909324408, 0.24939411878585815, 0.1734665185213089, 0.2923654615879059, 0.17090201377868652, 0.07970554381608963, 0.630560576915741, 0.4072439968585968, 2.48055362701416, 0.20415771007537842, -0.44723913073539734, -0.5324724316596985, 0.45582911372184753, 0.19513754546642303, 0.322083443403244, -0.12491130083799362, -0.10494505614042282, 0.833539605140686, 1.4451780319213867, 0.42491820454597473, -0.02818743884563446, -1.084568738937378, -1.0856214761734009, 0.12048393487930298, -0.17535515129566193, -0.029498199000954628, -1.0472146272659302, -0.06253772974014282, -1.8932443857192993, -1.1398168802261353, -0.7421925663948059, 0.6393010020256042, -0.5662417411804199, -0.9038388729095459, 0.3102126121520996, -0.06312230974435806, 0.957832932472229, 0.3540431261062622, 0.24079760909080505, -0.16631868481636047, 0.6837397813796997, -0.7159861922264099, -1.5155874490737915, 0.9188432097434998, -0.4650467038154602, 0.05570192635059357, 0.6328984498977661, 1.02590012550354, -2.90317702293396, -1.0468684434890747, 1.2337498664855957, -1.7055178880691528], [0.47626402974128723, -0.3627007007598877, -0.8921003341674805, -0.1311357170343399, -1.6965107917785645, 0.40589582920074463, 1.1765064001083374, -1.4324692487716675, -0.6737122535705566, 1.201277494430542, 0.3682346045970917, -0.3279290795326233, 1.4351848363876343, -1.6871209144592285, -0.2809986174106598, 0.08910386264324188, -0.3242110311985016, 0.7949813604354858, -1.6841487884521484, 0.0419415645301342, -0.7951081395149231, 1.1987022161483765, 1.0446423292160034, -0.7540777921676636, -1.110982060432434, -0.05243905261158943, -0.86948162317276, -0.5787657499313354, 0.42370519042015076, -0.4685525596141815, 1.8138983249664307, 0.059520427137613297, -0.5323241949081421, -1.1301554441452026, -0.5311712622642517, 1.1020461320877075, 0.5418751239776611, 1.4253020286560059, 1.3063513040542603, 0.5486820340156555, -0.7083648443222046, -0.16049906611442566, 0.0593281127512455, -1.6690553426742554, 0.3216547667980194, -1.0518375635147095, -0.09585031121969223, -0.5272992253303528, 0.5736538171768188, 2.6331241130828857, -1.2069087028503418, -0.027186580002307892, 2.0459628105163574, -0.938170850276947, -0.2334383726119995, -0.11366590857505798, 0.30666616559028625, -1.6295677423477173, 0.3029867112636566, 0.60346519947052, 2.554124355316162, -0.08022479712963104, -0.5807622075080872, 0.13223762810230255], [-2.6241302490234375, 0.33901074528694153, 1.565286636352539, -0.03310404345393181, -0.03400510922074318, -0.24262818694114685, 0.14173579216003418, -1.2457361221313477, 1.556157112121582, -0.45594120025634766, -0.4500073790550232, 1.6717227697372437, 0.03138326480984688, -0.41074836254119873, 1.6706821918487549, 0.19934625923633575, 0.9619471430778503, -0.5526202321052551, -1.7544792890548706, 0.19771327078342438, 1.4599751234054565, -0.43941211700439453, 0.3906843364238739, 0.7519165873527527, 0.29140812158584595, 0.10033553838729858, 0.8053041100502014, 0.2440894991159439, -0.5051175951957703, 1.0927962064743042, 0.8403662443161011, 0.41254448890686035, 0.7595425248146057, -2.6139895915985107, 0.305851548910141, 0.07747472077608109, -1.9600838422775269, 2.1217827796936035, 0.3910723030567169, -0.4022234380245209, -0.33886682987213135, 0.3079967498779297, 0.024515388533473015, -1.5770111083984375, -0.3604246973991394, 0.49143627285957336, 0.6596852540969849, 0.913526177406311, -1.0220931768417358, -0.5900863409042358, 0.4200451970100403, -0.5641858577728271, 0.26408806443214417, -0.7852447628974915, 0.6409056186676025, 0.11563713848590851, 0.6402919292449951, 0.09382452815771103, -0.7299577593803406, -1.164474606513977, 0.10823257267475128, -0.5488094091415405, -1.4282312393188477, -0.483622670173645], [0.8639504313468933, 1.3144090175628662, 0.8382378220558167, 1.4408756494522095, -0.29636409878730774, -0.07282564043998718, -0.8781378865242004, -0.6975635290145874, 1.6998779773712158, 0.5192322731018066, 0.7358409762382507, 2.7045629024505615, 2.023057222366333, -0.7820925116539001, 1.0720834732055664, -0.6707949638366699, 1.0838983058929443, -1.800173044204712, -0.8540792465209961, -0.8314402103424072, 2.106879711151123, -1.6960121393203735, 1.2090988159179688, 1.1745225191116333, 0.21123374998569489, 0.9305042028427124, 2.6903953552246094, 0.3472551703453064, -0.1859285682439804, 1.4567855596542358, 0.8817878365516663, -1.0671945810317993, 0.28148236870765686, 0.37875786423683167, -1.0624080896377563, -0.8689408898353577, -0.5779282450675964, -1.1256519556045532, 0.6099210977554321, 0.3916228711605072, -2.1584393978118896, -0.34197917580604553, 0.24037520587444305, -1.2846587896347046, -0.2018256038427353, 0.3723554313182831, -1.6439969539642334, -0.23576058447360992, -2.324458599090576, 0.8270865082740784, -0.6788032054901123, -1.4082980155944824, -1.813498616218567, -1.6186257600784302, 1.3463342189788818, -0.8091784119606018, -0.5387489199638367, -0.36283519864082336, -1.4947246313095093, 0.8040604591369629, -2.514575481414795, 0.9299538731575012, -0.7476893067359924, 0.062265776097774506], [0.31638234853744507, 0.864777147769928, -1.245194435119629, 0.6316952705383301, -0.042534977197647095, -0.1497233510017395, -0.279050350189209, 1.1402994394302368, 0.4842745363712311, 1.2972815036773682, 0.9357013702392578, -2.0091545581817627, -0.10172495245933533, 1.0270110368728638, -0.9157797694206238, -1.709349513053894, -2.3339133262634277, 2.0197603702545166, -0.9423668384552002, 0.16915524005889893, -1.0996613502502441, 0.4261535108089447, 1.680699348449707, -1.4839746952056885, -0.07135990262031555, -0.7865275144577026, -1.9219657182693481, -0.9180206656455994, 0.022351490333676338, 1.2539845705032349, 1.299833059310913, -0.02115975320339203, -1.6937263011932373, 0.6718358397483826, -0.008554019033908844, -0.01956520415842533, -1.1728698015213013, -0.162626251578331, 2.0049469470977783, 1.5216397047042847, 1.301401972770691, 0.6831285357475281, -0.40635454654693604, 0.0019021439366042614, 0.1545584350824356, 0.3991532623767853, -0.3562451899051666, -0.8836852312088013, 1.1446409225463867, -0.38013988733291626, 0.8623542189598083, -0.17686472833156586, 0.695436954498291, 0.5290508270263672, 0.760155200958252, 0.7070104479789734, -0.9656563997268677, -1.3841845989227295, -0.04374552145600319, 1.6943448781967163, -0.8474162817001343, -0.9452128410339355, 2.3303966522216797, -1.5767143964767456], [-0.5816718935966492, -0.41499564051628113, -0.6903476715087891, -0.6569643616676331, 1.1038422584533691, 0.2124847173690796, 0.18789586424827576, -1.6894078254699707, 0.8133772015571594, 0.1701420545578003, 0.15670719742774963, 0.8441293239593506, 0.3222537636756897, -0.026573948562145233, 0.4295389950275421, -0.7183917760848999, -0.9197143316268921, -1.3769526481628418, 0.9888535737991333, -0.6619219779968262, 0.18392722308635712, 1.1356672048568726, 1.847802996635437, -0.3610532581806183, 0.1748092919588089, 0.10296069085597992, -1.32536780834198, -0.4818105101585388, 2.8531901836395264, -0.057821985334157944, 1.8103140592575073, 0.784964382648468, 1.798912763595581, 1.359023928642273, 0.23863103985786438, -1.2540137767791748, 1.4669009447097778, 0.7286076545715332, 0.5011237859725952, 0.8914228677749634, -0.801724374294281, 0.1388131082057953, 0.002337648067623377, -1.5085853338241577, -1.217527985572815, -0.12283164262771606, -2.2543201446533203, 0.753483235836029, 0.7880116105079651, -0.0745398998260498, 0.2870529890060425, -1.8507272005081177, 0.17429620027542114, 0.7635998129844666, 0.5131475329399109, 0.9346216320991516, -0.06840269267559052, 2.0264604091644287, 0.43739089369773865, 0.8038463592529297, 1.0973892211914062, -0.7842851281166077, -1.14949631690979, 0.790923535823822]], [[-1.279953956604004, -0.6160572171211243, 0.9996698498725891, -1.1180498600006104, -1.004956603050232, -1.269560694694519, -1.0677045583724976, 1.9004151821136475, 0.5517888069152832, 0.9722903966903687, -1.9816550016403198, -2.457585096359253, -1.179125189781189, 1.0562944412231445, 1.1865676641464233, -2.0245754718780518, -1.791841983795166, 0.6510434746742249, 0.2402716428041458, 0.4296501576900482, 0.6572490334510803, -1.046489953994751, 0.10760582983493805, -0.30710411071777344, 1.2317569255828857, -0.33581119775772095, -1.947200894355774, 1.2642539739608765, 1.8175526857376099, 0.9674302935600281, -0.3589092791080475, -0.9893718361854553, 0.9997820854187012, -0.5646253824234009, 0.5054417848587036, 0.7701002955436707, -0.16397438943386078, -2.0658774375915527, 0.4813714921474457, -1.049729824066162, -1.6199027299880981, -0.07810775190591812, 0.10766378045082092, -0.12551642954349518, 0.6125551462173462, -1.9776705503463745, 0.5594844818115234, 1.6242954730987549, -0.11603474617004395, -0.5438166856765747, -0.20839431881904602, 1.215266227722168, -0.22255584597587585, -0.8388031721115112, -1.7559230327606201, 1.4067758321762085, 1.4250935316085815, -0.019607912749052048, 2.085129737854004, -0.4282550513744354, 0.7128222584724426, -1.3589022159576416, -1.0492678880691528, -0.03090570494532585], [0.4628725051879883, 0.6063375473022461, -0.9331610202789307, -0.5591902136802673, -0.13281725347042084, -0.30639317631721497, -0.2832806706428528, 0.8107817769050598, 1.165527105331421, -1.8971138000488281, 1.323367714881897, 0.1668599396944046, 0.3179636001586914, 0.3290240466594696, 0.9359439611434937, 0.6962593793869019, -1.5688550472259521, 0.8707560896873474, -1.1553701162338257, 0.3822036683559418, 0.21011683344841003, -1.5739116668701172, 0.00763573357835412, -0.2656339704990387, -0.688567578792572, 2.055619239807129, 0.1570737510919571, -0.38075757026672363, 1.765931248664856, -0.035051796585321426, -0.14131540060043335, -2.0396265983581543, 0.6059733033180237, 1.0125623941421509, 1.936902403831482, -1.0317901372909546, 1.0567675828933716, 1.4211300611495972, 0.7312470078468323, 0.0004376191063784063, -1.6787775754928589, 0.1879522204399109, -0.5667224526405334, -1.7716076374053955, 0.6081501841545105, -0.12629400193691254, -0.24238882958889008, 1.999988079071045, -0.5465682744979858, -2.113917350769043, -2.260662317276001, 0.5262123346328735, -0.2282325029373169, -0.7765992283821106, -2.1633615493774414, -0.8526902794837952, -1.1533225774765015, -0.12701547145843506, -0.8885976076126099, -0.9055554866790771, -0.06567306071519852, -0.8213518857955933, 0.30681097507476807, -3.4890921115875244], [-1.283579707145691, 1.2037767171859741, 0.1792767196893692, -1.1309115886688232, -0.219748392701149, -0.6168520450592041, -0.4062117338180542, 1.4187238216400146, -0.4551678001880646, 0.17878110706806183, -0.8977140784263611, -0.788196325302124, -1.6043168306350708, -0.7006797194480896, -0.4537413716316223, -0.3494471311569214, 0.01974780112504959, 0.061055779457092285, 0.6308714747428894, -0.2514100968837738, 1.2957452535629272, 0.5811610817909241, 1.3666226863861084, -0.08464314788579941, 1.1991623640060425, 0.06524880975484848, -1.7045265436172485, -0.05154397338628769, -0.20740807056427002, -1.5042614936828613, -1.028358817100525, -0.6435770988464355, 2.279287099838257, 0.28586164116859436, 1.8088774681091309, 0.9187138080596924, 1.070163369178772, 0.797220766544342, -1.5280883312225342, 1.2445708513259888, -1.035203218460083, 0.42310360074043274, -0.12211237847805023, 1.6191723346710205, 1.192905068397522, 1.8358203172683716, 0.4801208972930908, -0.00454776082187891, 0.7659798264503479, 2.4078879356384277, 0.5153617858886719, 0.03587067499756813, -1.2473448514938354, -0.760235071182251, -0.09207724034786224, 0.6738526821136475, 0.620242714881897, -0.1053115651011467, 0.12852054834365845, 1.5394366979599, 1.8534750938415527, 1.2020151615142822, 1.5357857942581177, 1.2501819133758545], [1.8346267938613892, -1.266554355621338, -0.5137951374053955, 0.23736003041267395, 0.39106374979019165, 1.3472763299942017, -1.8483145236968994, 0.15010730922222137, 0.269502192735672, -0.6765498518943787, -0.13676701486110687, 1.0100003480911255, 0.47231218218803406, -0.12119347602128983, 0.2133304625749588, -0.11790558695793152, 0.9489354491233826, -0.09370438754558563, 0.9949321746826172, -0.5499192476272583, -0.023564215749502182, -1.11411452293396, -0.19425441324710846, 0.8555719256401062, -1.0062843561172485, -0.33976125717163086, 0.11938902735710144, -0.3426598906517029, 0.8570409417152405, -1.2319366931915283, 0.8793560266494751, 1.344630241394043, 1.6850330829620361, -0.046261727809906006, -0.7885118722915649, -1.566705584526062, 0.7056165933609009, 0.8463314771652222, 0.31323671340942383, 0.3673610985279083, 0.6974493861198425, -0.341010183095932, -1.45907461643219, -1.4159059524536133, -0.7706493139266968, -0.9017643332481384, -1.5783023834228516, -0.41286250948905945, -1.0220776796340942, 0.6832398772239685, -0.05107981339097023, 0.8219492435455322, -1.311927318572998, -1.0842654705047607, -1.5840054750442505, 0.3100616931915283, 3.1597750186920166, 0.24542008340358734, -0.4130656123161316, 0.23803766071796417, -0.007701708935201168, 0.9950780272483826, 0.01981261931359768, -0.8630015254020691], [-1.8907221555709839, 0.38540878891944885, 1.0339497327804565, 0.17416469752788544, -0.8695354461669922, -0.41264787316322327, -0.30134317278862, 0.609575629234314, -0.0373137928545475, 1.2036628723144531, -0.1780203878879547, -0.6263366341590881, 1.8862498998641968, -0.983154296875, -0.7684646248817444, 2.1231918334960938, 0.3077166676521301, 1.8227672576904297, -2.4914472103118896, 1.2892688512802124, 0.024232281371951103, 0.1291380077600479, 0.5211510062217712, -2.7540884017944336, -0.880375325679779, 0.4748156666755676, 3.280484437942505, -0.734463095664978, 2.4022715091705322, 0.6666901707649231, 1.0606590509414673, -0.9445372223854065, 0.6408244371414185, -0.8412412405014038, -1.19634211063385, 1.3122743368148804, -0.6845633387565613, -0.4900459945201874, 0.7229471802711487, -0.7951647043228149, -1.2512199878692627, 0.25068992376327515, 0.29657167196273804, 1.5548866987228394, -1.1132643222808838, 0.7431970238685608, 0.741662859916687, -0.4214533269405365, -0.14950981736183167, 0.04107408598065376, -1.0215051174163818, -1.315905213356018, 0.7301647663116455, -1.0596551895141602, -0.9686294794082642, 0.8134691715240479, -0.534144401550293, -0.3964405953884125, -1.4020283222198486, -1.2373522520065308, -1.3910726308822632, -1.3860958814620972, 1.258571743965149, -0.9760587811470032], [-2.3431038856506348, -1.1263753175735474, 0.6967557072639465, -1.1906239986419678, -0.0696740373969078, -0.9768961071968079, -1.8019131422042847, -0.5871549248695374, -1.0703202486038208, 0.13781099021434784, 0.6738025546073914, -0.10938538610935211, 2.619720220565796, -1.110555648803711, 1.3088946342468262, 0.38069814443588257, 1.367080569267273, 0.49181103706359863, -2.3132083415985107, -0.3493176996707916, 0.9929517507553101, -0.02758386917412281, -1.3802120685577393, -0.6042711734771729, 0.2782595455646515, 1.1835925579071045, -0.662032961845398, 0.04236043244600296, -1.0001064538955688, -1.2735131978988647, -0.5881301164627075, 1.2959563732147217, 0.6613765954971313, -0.3469172716140747, -0.09621936827898026, 0.46980953216552734, -0.19815851747989655, -0.5142219662666321, -0.38081440329551697, 0.5668272376060486, -0.4785928726196289, 0.7067695260047913, -1.961521863937378, -0.854141354560852, -0.5196807384490967, 0.5035271048545837, -1.021882176399231, -0.6350343227386475, 0.22382725775241852, -0.20150126516819, -0.7761430144309998, -0.4345782399177551, -1.0312949419021606, 0.7105249762535095, 0.3398021459579468, 0.7663951516151428, 2.173912763595581, 1.1747143268585205, 0.4217837452888489, -0.16563311219215393, 2.218447208404541, 0.19466865062713623, -0.25797784328460693, 1.2692523002624512], [0.4117867648601532, -0.26381590962409973, 0.44421863555908203, 1.9802014827728271, 0.166892871260643, 0.41853103041648865, -0.8326524496078491, -0.41872110962867737, 0.9571453928947449, -0.021325645968317986, 0.19744333624839783, 1.183922529220581, -0.2429741770029068, 1.6476994752883911, -0.555851399898529, -0.6657913327217102, -0.7138451337814331, -1.4098362922668457, 0.6448240876197815, 0.11049598455429077, 1.4621315002441406, -0.6484695076942444, 0.8371844291687012, 1.4197953939437866, -0.14677178859710693, -0.09642885625362396, -2.4290921688079834, 2.00850510597229, -0.06405671685934067, 0.7521620988845825, -0.46113643050193787, 0.4048464596271515, -1.095330834388733, 0.807016909122467, 0.15720871090888977, -0.31606048345565796, 0.48843416571617126, -1.7440528869628906, 0.21442176401615143, -0.3899080455303192, 1.4976296424865723, 0.8168826103210449, -0.14408200979232788, -0.25363150238990784, 3.027841091156006, -0.9625516533851624, 1.9403542280197144, 0.9593336582183838, 0.8167067766189575, -1.8807340860366821, 0.861061692237854, -0.4763568937778473, -0.8841438889503479, 0.6219140291213989, -0.6908262968063354, 0.5945615768432617, -0.9252094626426697, 0.6085829734802246, 0.4558362066745758, -0.8358479142189026, -0.9486767053604126, -0.6734093427658081, -0.5301512479782104, -0.6025046110153198], [1.852903127670288, 0.4144555926322937, -0.3521604537963867, -0.3363032341003418, -0.8189142346382141, 0.2642752230167389, -0.07895781099796295, -0.6958801746368408, -0.7114008069038391, -0.4240822196006775, -1.077291488647461, 0.09817306697368622, -0.930112361907959, 0.5227359533309937, -0.3476731777191162, 0.5173784494400024, -0.5966096520423889, -1.88602876663208, -1.3306903839111328, -0.30026423931121826, 0.3775738775730133, 0.40221843123435974, 0.5155473351478577, 0.6871378421783447, 0.36985957622528076, 0.4444774091243744, -0.21929647028446198, 0.5644184350967407, -0.10225122421979904, -0.06550837308168411, 0.1972229778766632, 2.3073863983154297, -1.4591786861419678, -0.46659308671951294, -0.25178998708724976, 0.4394550919532776, 0.030403653159737587, 1.6543138027191162, -1.5394114255905151, 1.8360337018966675, -1.2312308549880981, -0.6932327747344971, 0.5829170346260071, 0.10989007353782654, 0.0028024918865412474, 0.1010860726237297, 0.41663408279418945, 0.990801215171814, 0.4378989636898041, -1.2280985116958618, -0.7072792053222656, 1.4000340700149536, 1.9535826444625854, 0.9641135931015015, 1.0990877151489258, 0.3367008864879608, 2.1004538536071777, -0.5456416606903076, -0.30883529782295227, -0.7497166395187378, 1.592954397201538, -0.49618202447891235, 0.5355872511863708, -1.3538793325424194], [-1.4378212690353394, 1.0183064937591553, 0.7379837036132812, 0.878753125667572, 0.23666289448738098, -0.2587861716747284, -0.3739846348762512, -1.0723695755004883, -0.34636470675468445, 1.3794361352920532, -0.12160094082355499, -1.2731781005859375, -0.20390133559703827, -0.9462082386016846, -1.1088943481445312, -0.07637085020542145, -0.7296434640884399, -1.2250570058822632, 1.5490106344223022, 1.7468217611312866, 0.17093303799629211, 1.5040730237960815, -0.4287012815475464, -0.0072801001369953156, 1.3300824165344238, 0.3224194347858429, 0.8401212096214294, -2.7249093055725098, 0.3085741102695465, -0.4050181806087494, -0.09118630737066269, -1.5000832080841064, -0.7580959796905518, 0.6101497411727905, 0.5960333347320557, 0.7825926542282104, 0.85992032289505, 1.1054673194885254, 0.3939518630504608, -0.22872893512248993, -0.5660899877548218, -0.41188210248947144, -0.9606953263282776, -0.9880048036575317, -0.08396982401609421, -2.273559093475342, -1.2232555150985718, -0.7990161180496216, 0.5299466848373413, 1.0802807807922363, 0.33431708812713623, -0.03870897367596626, -0.48555073142051697, 1.118587613105774, 0.7146350145339966, -0.6497265696525574, -0.017248759046196938, 0.4223219156265259, 0.9239057898521423, -0.3174450993537903, 1.5150575637817383, 1.1879867315292358, -0.40283674001693726, -0.17259639501571655], [-0.8989632725715637, -0.8745676279067993, 0.34387657046318054, 1.4575897455215454, 1.4731487035751343, 0.2334730476140976, -0.32460805773735046, -0.594416618347168, 0.48383259773254395, -0.4040856659412384, 0.772034227848053, -0.05897550657391548, 0.355744332075119, -0.45271843671798706, 2.0842161178588867, 1.2032252550125122, 0.15742062032222748, -0.8073006272315979, 0.633104145526886, 0.824955940246582, -1.0576523542404175, 0.14946265518665314, 2.1114773750305176, 0.2559845745563507, -0.3155668377876282, 0.6699381470680237, 1.8983629941940308, -0.35976630449295044, -0.3484162986278534, -0.1523588001728058, 0.16521680355072021, 1.2786155939102173, 0.42656999826431274, -2.1184041500091553, 0.5315694212913513, 0.4979761242866516, 0.7367697358131409, 0.37691470980644226, -0.020146511495113373, 0.6139426231384277, -0.5744167566299438, -1.5768693685531616, -0.2356051802635193, -0.9987313151359558, 1.7751117944717407, -0.02259250357747078, 1.6441134214401245, -1.2590882778167725, -1.1267911195755005, 0.5007064342498779, -1.6315810680389404, -0.5709758400917053, -1.868010401725769, 0.4814133644104004, -1.660117745399475, -0.08186803758144379, -1.711890697479248, 0.5296099781990051, 1.1102181673049927, 0.16424964368343353, -0.9433190822601318, 0.46220409870147705, 1.0122227668762207, 0.7989479303359985], [1.2946593761444092, 1.9302805662155151, 0.2597943842411041, -0.22692182660102844, -0.6023266315460205, 0.018776608631014824, -0.20093780755996704, -0.499337375164032, 0.36964184045791626, 2.00644850730896, -0.7636719346046448, -0.45962753891944885, -0.331974059343338, -1.0034511089324951, 1.0679951906204224, -0.13810276985168457, 0.6084507703781128, -1.5930707454681396, 0.5707660913467407, -0.11970788985490799, -1.4413307905197144, 0.8660524487495422, 0.9487757086753845, 0.2321598380804062, 0.47581371665000916, -0.9398449659347534, 1.0351611375808716, -0.22718358039855957, 0.48386818170547485, 0.6052852272987366, 0.4902229905128479, -0.7192102670669556, 0.28369516134262085, 0.15395978093147278, 1.1887308359146118, -0.7401849627494812, 0.9670329689979553, 0.4880593419075012, 0.0675697848200798, 1.8977575302124023, 0.8222788572311401, -0.5326683521270752, 0.3140498995780945, -0.7374113202095032, -1.153651237487793, -0.48984450101852417, -0.4775482416152954, 1.7176231145858765, 0.810215175151825, -2.4088127613067627, -0.14535285532474518, -0.6888182759284973, -0.13287459313869476, 0.1854962557554245, 0.04844951257109642, 0.4300859868526459, -0.7742994427680969, 1.4190995693206787, -0.12620404362678528, 1.237034559249878, -0.09160531312227249, 0.5015392899513245, -0.9523482322692871, 1.1138787269592285], [0.29490765929222107, -0.39645102620124817, 0.38532671332359314, 0.4753826856613159, -1.5354543924331665, -0.7465856075286865, -0.9922381639480591, 0.8310493230819702, 0.18762171268463135, -0.6644507050514221, -0.7077614665031433, -0.039350010454654694, -2.2990148067474365, -0.16877107322216034, 1.5640983581542969, -1.4035494327545166, -1.1968687772750854, -1.1314923763275146, 0.4406205415725708, -0.7534507513046265, 1.870538592338562, -0.20240260660648346, -0.49794867634773254, -1.1012840270996094, 0.6805511116981506, 1.071608543395996, 0.17080950736999512, 1.3044053316116333, 1.242796540260315, 0.3613174557685852, 0.06728771328926086, -0.5427314639091492, 0.22998665273189545, -0.33789461851119995, 1.489331841468811, 0.767981767654419, -0.15290775895118713, -0.3733164966106415, 0.43045011162757874, 1.0660746097564697, 0.30078452825546265, 1.0407698154449463, -0.42374515533447266, 0.2492169290781021, 0.678574800491333, 0.2271493375301361, 0.4186370074748993, 1.0504661798477173, 0.03492581099271774, 1.037218451499939, -0.4703746438026428, 0.5396657586097717, -0.3772940933704376, -0.32690781354904175, 0.655674934387207, 1.3336074352264404, -0.058609090745449066, 0.16296041011810303, 1.394927978515625, -0.7243450284004211, 0.9893326759338379, -1.0811567306518555, -1.8002487421035767, 0.4312465190887451], [-0.20289508998394012, -0.42567312717437744, -2.3121469020843506, 0.4457888603210449, -0.38042548298835754, -1.0975983142852783, -0.03929280489683151, -1.119173288345337, -0.7006880640983582, -0.743288516998291, 1.036751627922058, -0.0502912774682045, 1.3461846113204956, -0.7511456608772278, -1.1358426809310913, 1.8109080791473389, -1.326167106628418, 1.2654954195022583, -1.0491005182266235, 0.7671260833740234, -0.6859959959983826, 1.5593405961990356, 2.025953769683838, 1.0971591472625732, 0.4107394814491272, -0.30582094192504883, 0.8485366106033325, 2.1631383895874023, -0.38581642508506775, -1.0673532485961914, 1.855400562286377, -0.15129543840885162, 0.8064190745353699, 0.2669338285923004, 0.34848177433013916, -0.1976155787706375, -0.056623492389917374, -1.589194893836975, -0.4844615161418915, -0.7823276519775391, 1.2475695610046387, -0.22885139286518097, 0.19063137471675873, -1.5898244380950928, 0.15996578335762024, 1.609471321105957, 0.2534945607185364, 0.9643060564994812, -0.22611163556575775, -0.5263950824737549, 1.046466588973999, -0.3636203408241272, -0.9595946073532104, -0.015127908438444138, -0.29786303639411926, 1.1378110647201538, 0.8100088834762573, -1.3530939817428589, 0.596145510673523, -1.2496598958969116, 0.2651059925556183, 0.6556275486946106, -0.4312736392021179, -0.4396377205848694], [0.8494310975074768, -0.21529564261436462, -1.0392838716506958, 1.7186962366104126, 0.9644219279289246, 0.5621758699417114, 0.4356314241886139, 1.1098506450653076, 0.17943821847438812, 0.8839022517204285, 0.42547059059143066, -0.5741647481918335, 1.1636534929275513, 0.2796519100666046, -0.6976014971733093, -0.3311852514743805, -2.2853822708129883, -1.5536330938339233, -1.348146915435791, 0.7956990599632263, 0.762464165687561, 1.2402788400650024, 0.5089648365974426, 1.8096579313278198, -0.37223732471466064, -0.6800718307495117, -0.8567442297935486, 0.26151347160339355, -0.5864918231964111, -0.8570237755775452, -0.4819689989089966, 0.20421355962753296, -1.241112232208252, 0.03054048679769039, 0.018350427970290184, 1.7065799236297607, 0.18260003626346588, -1.2554631233215332, 0.28380337357521057, 0.2198420614004135, -1.2326818704605103, -3.0200252532958984, 0.7750603556632996, -0.16169552505016327, -0.6993016600608826, -0.15083551406860352, 1.3017535209655762, 1.7662708759307861, -0.4445127844810486, 1.545525074005127, -0.029086990281939507, 0.02635383978486061, -0.07475262880325317, -0.04334151744842529, -0.12689724564552307, -0.6685713529586792, 0.03407662361860275, -0.9958080649375916, 0.577201247215271, -0.6321569681167603, 0.5038849115371704, -0.10716787725687027, -1.7749274969100952, 1.1907894611358643], [0.30621784925460815, -0.05482300743460655, -0.32525354623794556, 1.8049708604812622, -0.23402263224124908, 0.9681760668754578, 0.21120670437812805, -1.3210361003875732, 0.728491485118866, -1.5151286125183105, 0.4537009000778198, 0.41374462842941284, 0.2375398874282837, -0.7435579299926758, 1.397802472114563, 1.7701517343521118, -0.9155606031417847, -0.38045865297317505, 1.6291191577911377, -0.4323292672634125, 0.961184561252594, -0.9784068465232849, -0.633400559425354, 0.26013097167015076, 0.45214128494262695, -1.4457045793533325, -1.0499062538146973, 0.5112688541412354, 1.2757476568222046, 0.5147407054901123, 0.35967937111854553, 0.43580010533332825, -1.2190502882003784, 0.3962579369544983, 0.8991712927818298, 0.5954474210739136, -1.166929006576538, -0.958499014377594, 1.1200820207595825, -1.384940266609192, -1.0681498050689697, 1.583078384399414, -0.9018523693084717, -0.7317481637001038, 0.6386840343475342, 0.7848655581474304, -0.16740551590919495, -1.4323614835739136, -0.5178028345108032, 0.16304154694080353, -0.9197327494621277, -0.9756007194519043, 1.2500271797180176, 0.5410477519035339, 0.7814403176307678, 0.8961166143417358, -1.0051637887954712, 0.4836586117744446, 1.1929501295089722, 0.679314911365509, -1.1566542387008667, -1.5547057390213013, -0.803153395652771, 0.5467050075531006], [-0.18544551730155945, 0.278155654668808, -1.2629302740097046, 0.8317814469337463, -0.7319740653038025, 1.1288713216781616, -0.02692096121609211, -2.983304738998413, -0.20099715888500214, -0.8784583210945129, -0.8480167388916016, -0.16243065893650055, 1.0687962770462036, 0.843956470489502, 1.3950250148773193, -0.006775322370231152, -1.8736416101455688, -0.5627657771110535, -0.9102773070335388, -0.6916677355766296, 2.660832643508911, 0.7955988645553589, -0.9716129899024963, 1.1903032064437866, -0.2986043393611908, -1.3516161441802979, 1.168671727180481, -0.479875773191452, 0.12517546117305756, 2.297947645187378, -1.0188980102539062, 1.0360150337219238, -0.31614920496940613, -1.1174938678741455, -0.25544971227645874, 0.17025402188301086, 0.18396884202957153, -1.5423492193222046, 0.8352594971656799, -0.8572018146514893, 0.3042236566543579, 0.20090848207473755, -0.5590395331382751, 0.5946524143218994, -0.7217796444892883, -0.949777364730835, -1.4248555898666382, -1.867972493171692, -0.7026321887969971, -2.198394775390625, -0.3678354322910309, -0.5476751327514648, 1.6355128288269043, -1.1710472106933594, 1.1364130973815918, 0.3579765260219574, 0.28504258394241333, -0.6431961059570312, -0.8386884927749634, -0.33704936504364014, -0.0642470046877861, 0.5602267384529114, -0.5295238494873047, 1.5474653244018555]], [[1.6693686246871948, -2.483076333999634, 0.12001407146453857, -0.13005711138248444, -0.9967870116233826, -2.704066038131714, -1.7473961114883423, -0.2014160305261612, 0.6785237789154053, 0.4372639060020447, 0.5011740922927856, -0.23343618214130402, 1.0114980936050415, 0.4695163667201996, 1.1200000047683716, -0.14076656103134155, -0.665596067905426, -1.5401561260223389, 1.5397344827651978, -0.5880365967750549, -1.0492141246795654, -2.1425108909606934, 0.5628193020820618, -1.3827499151229858, 0.0653817355632782, 0.2795105278491974, 1.4245837926864624, -0.7208463549613953, -1.949139952659607, 0.8661922216415405, 0.2561570405960083, -0.6391749978065491, -0.03710775822401047, 1.8843717575073242, 0.05465118587017059, -0.6617418527603149, -0.08113501965999603, 0.5789934396743774, 0.36482372879981995, 0.33901694416999817, 0.32801559567451477, -1.1705477237701416, 0.6045888066291809, 1.042079210281372, 0.6626666784286499, -0.06385974586009979, -0.07897458970546722, 0.19271844625473022, 0.06323280930519104, -1.649280309677124, -0.12423093616962433, 1.2666898965835571, -0.3213159441947937, -0.5224445462226868, 0.6217910051345825, 0.09953209012746811, -0.6863774657249451, 0.18987660109996796, 1.7299407720565796, 1.8567564487457275, 0.17497456073760986, -0.14183993637561798, 1.2208938598632812, 0.4903475344181061], [0.574055552482605, -0.03750361502170563, -1.276310682296753, -1.4215664863586426, -0.05281725898385048, -1.065730094909668, 0.8345025181770325, 0.9644421339035034, 1.5105040073394775, 0.14101266860961914, 1.9865413904190063, 1.3150159120559692, 0.42888861894607544, 0.80147784948349, -0.5164341330528259, -3.118182897567749, 1.0983068943023682, 0.5442060828208923, 1.736991286277771, 0.5411150455474854, -0.12752681970596313, -0.4405460059642792, -0.5650120377540588, 0.21583804488182068, -1.7724534273147583, -1.0814930200576782, 0.8762532472610474, -2.0076615810394287, -1.5287182331085205, -0.5604720711708069, -0.37819704413414, 0.09841744601726532, 0.8811007142066956, 0.31697705388069153, 0.623593270778656, -0.559839129447937, 0.051346153020858765, -0.6860194206237793, -0.08713014423847198, 0.6069824695587158, 0.5255100131034851, -1.87220299243927, -1.1198173761367798, 2.0182058811187744, -2.0275206565856934, 0.08992525190114975, -1.149557113647461, -0.192964568734169, -0.5637987852096558, 0.09119017422199249, -1.088201880455017, 1.069815993309021, 2.21403169631958, -2.4384334087371826, -0.8260675072669983, -1.5027443170547485, 0.14151571691036224, 1.4273366928100586, 1.4152957201004028, 1.3436318635940552, 0.24831610918045044, -0.536372184753418, 1.1526192426681519, -0.592350423336029], [-0.7679307460784912, 0.08941148966550827, -1.220232367515564, -0.16935715079307556, -0.920637845993042, 0.12135239690542221, 0.992047905921936, -0.5344865918159485, 0.4972158968448639, 0.9880134463310242, -0.7551485896110535, -0.049712300300598145, -0.3396786153316498, 1.8394063711166382, -0.4816047251224518, 0.14651602506637573, -1.3879245519638062, -0.9070892930030823, 0.6201841831207275, -0.26400578022003174, 0.7687034606933594, 0.09005006402730942, 2.6467742919921875, -0.21276643872261047, 0.9800443649291992, -0.6944661140441895, -0.4127323031425476, -1.5103859901428223, 2.6688411235809326, -2.077820062637329, 0.14751334488391876, -0.41501426696777344, 0.8284316062927246, -0.5147935152053833, 1.9319367408752441, 1.176098108291626, 0.5752525925636292, 0.5455314517021179, 1.1120167970657349, 0.4912285804748535, 1.659765362739563, 0.9524226188659668, 0.030035853385925293, 0.5216901302337646, 0.2936675250530243, 1.282731056213379, -0.16396719217300415, 0.20250225067138672, -1.1694313287734985, 0.7778228521347046, 0.34858885407447815, -0.5647812485694885, 1.8266050815582275, 0.13034836947917938, -1.0519931316375732, 1.2555075883865356, -0.0755084976553917, -3.0023438930511475, 0.45173877477645874, 0.08945809304714203, 0.18226958811283112, -0.10465089976787567, -0.7131412625312805, 0.22052015364170074], [-1.5977740287780762, -0.08372026681900024, 0.9634135961532593, 1.0978418588638306, -0.40297722816467285, 2.037614583969116, 0.40261754393577576, -0.3468180298805237, 0.25993403792381287, -0.5926457047462463, 0.9643853306770325, -0.8455010056495667, -0.6122591495513916, -1.0585112571716309, -0.7160094380378723, -0.18803143501281738, 1.9595186710357666, 0.33877381682395935, 1.2755241394042969, 0.1449190229177475, 1.0415313243865967, -0.9555771350860596, -1.035093069076538, 1.9607577323913574, 0.4695088565349579, -1.4889787435531616, 0.6967544555664062, -0.10883867740631104, -0.18226486444473267, 0.9165145754814148, -1.1422555446624756, 1.3184292316436768, 0.3490830063819885, -0.39783433079719543, -1.265401840209961, 1.844338297843933, -0.28609588742256165, -1.302280068397522, -0.07707741856575012, 0.5476939082145691, 0.7013795375823975, 1.5576955080032349, 0.636326253414154, 0.01624738797545433, 2.320136785507202, 2.0990424156188965, -1.0546531677246094, 1.0578608512878418, -0.5170668959617615, 1.583986759185791, -0.8697493076324463, 0.38141393661499023, 1.6703126430511475, 0.4210970401763916, 1.1836241483688354, -0.6241598725318909, -1.164780855178833, -1.099281668663025, 0.6985853314399719, -0.5676637291908264, 1.5615769624710083, -0.2562851309776306, 1.438626766204834, 0.43415161967277527], [-1.008139967918396, 0.1386002153158188, 0.5796401500701904, 0.5829049944877625, -0.014744335785508156, 1.2071722745895386, 1.0491958856582642, 1.0417866706848145, -0.6953059434890747, -2.0423152446746826, 0.6714187860488892, -0.7508044838905334, -0.4588521122932434, -0.16163498163223267, 0.4102363586425781, -1.677219033241272, -0.04861164093017578, 1.4559720754623413, -1.3821014165878296, -0.21119609475135803, 1.5442267656326294, 0.9272860884666443, 0.7106500864028931, 0.24335119128227234, -1.3469922542572021, 0.20281638205051422, -0.8722490668296814, 0.21567144989967346, -0.6804844737052917, -1.0944935083389282, 0.08393443375825882, 1.1605384349822998, -0.4103006422519684, 0.27684521675109863, 0.813885509967804, 0.6065249443054199, -1.1890093088150024, -0.4749634265899658, 0.4454396367073059, 0.11049844324588776, 1.8578436374664307, 0.679482102394104, -0.8042291402816772, 0.7697632908821106, 1.2138804197311401, 0.7421479821205139, 0.5988601446151733, -1.3343217372894287, 0.026255056262016296, 0.682756245136261, -0.310396671295166, -0.2561720609664917, 0.5350619554519653, -0.5647339820861816, -0.8186501264572144, -0.07931062579154968, -0.016269981861114502, 0.9288008809089661, -0.0957370474934578, -1.0884650945663452, 0.5529963374137878, -0.7272505760192871, 0.8126926422119141, -0.508187472820282], [0.6492453813552856, 0.3831597566604614, -0.1258544623851776, -1.2019609212875366, -0.3282935619354248, 1.4864698648452759, 0.2596432864665985, 1.2317891120910645, 0.1379862129688263, 0.07829545438289642, -0.9018182754516602, 0.4542944133281708, -0.7724809646606445, -0.7655104994773865, 1.111363410949707, -0.9317512512207031, 2.370243549346924, 0.06538983434438705, -0.49641308188438416, 1.4049818515777588, -0.9127324819564819, 0.1981116086244583, 0.10118179023265839, 1.0183123350143433, 1.798588514328003, 0.4112475514411926, -1.225598692893982, -1.261889934539795, -0.42825639247894287, 0.8730645775794983, 0.1969636082649231, 0.3364540934562683, -0.2508205473423004, 1.4187977313995361, -1.0688376426696777, 1.990577220916748, 1.26265287399292, 0.7657533884048462, 0.039348162710666656, -0.23543761670589447, 0.05939410254359245, -0.48150646686553955, -0.36307594180107117, -0.7521176934242249, 0.8223510980606079, 0.15944361686706543, 0.501099705696106, -0.6747158169746399, -1.1779736280441284, -0.8672982454299927, 0.2819685637950897, -1.1254199743270874, 0.6416962146759033, -0.8955376148223877, -0.4036557972431183, 0.2831224799156189, -1.1471447944641113, -2.0295753479003906, 1.191953420639038, -0.1358472853899002, -0.6067324280738831, 1.6278566122055054, 2.2002360820770264, 0.05231134220957756], [0.341178834438324, 2.657649278640747, -0.5864152908325195, 1.229604959487915, 1.0042905807495117, -0.5409359931945801, -0.9458861947059631, -0.7368704676628113, 0.17119915783405304, 0.37926244735717773, 2.640115737915039, -0.9913631081581116, 0.2300710380077362, -0.7514102458953857, 0.9043951630592346, 0.46724733710289, -1.2744417190551758, -0.4049414098262787, 0.10765296220779419, 0.772471010684967, 0.5813251733779907, 0.7263111472129822, 0.07743563503026962, 0.6570204496383667, 0.13751058280467987, 1.0316225290298462, 0.6429873108863831, -0.8199806809425354, 0.015138978138566017, -0.7021170258522034, -0.05452507361769676, 0.632208526134491, 0.16194570064544678, -0.04425327479839325, -0.5379634499549866, 1.5462924242019653, 0.07825995236635208, -0.8941624760627747, -0.4053542912006378, 1.0155236721038818, 2.0947463512420654, 1.3427225351333618, -0.2598535716533661, 0.9273083806037903, -1.5896728038787842, 0.9028159379959106, -0.1547987461090088, 0.337981641292572, -1.156127691268921, 0.4501473307609558, -1.1998841762542725, -1.5599243640899658, 1.5622144937515259, -1.1675564050674438, 0.9930763840675354, -0.4901351034641266, -0.039744727313518524, 0.06694691628217697, 1.1796844005584717, 0.5012404322624207, -0.8121668100357056, -0.44964227080345154, 1.3879607915878296, 1.4151084423065186], [-0.4072864353656769, 0.9538639187812805, -0.005431880243122578, 1.0714361667633057, -0.9404116868972778, 0.8740315437316895, -0.4989400804042816, -1.2197439670562744, 0.6629244685173035, 0.6253814697265625, -1.4380460977554321, -2.2581748962402344, -0.06952077895402908, 0.4943099915981293, -0.48744890093803406, 0.13446544110774994, -0.3825136721134186, -0.5112877488136292, 0.4277516305446625, 1.2649694681167603, 0.49866095185279846, -0.28250616788864136, -0.6538451910018921, 0.23434822261333466, -2.171501398086548, 0.1375926434993744, -0.08335935324430466, 0.006405659485608339, -0.24392864108085632, -0.17330209910869598, -1.4279048442840576, 0.6775554418563843, -0.29543396830558777, 0.038633182644844055, -0.4739074409008026, -1.4527223110198975, 0.6427724957466125, -0.19842591881752014, -0.23208148777484894, 0.40083184838294983, 1.7148016691207886, -0.01067575067281723, 0.5707886815071106, 1.8718814849853516, -0.27517208456993103, 1.247967004776001, 0.07521532475948334, 1.0412615537643433, 0.15740154683589935, 0.9957903027534485, -0.701547384262085, -0.4429342448711395, -0.573255717754364, -0.25896376371383667, 1.141019344329834, 0.47779518365859985, 0.7084728479385376, -0.16866256296634674, 0.1391945481300354, 0.46999624371528625, 0.408950537443161, -0.5055137276649475, -0.33724498748779297, -1.0063786506652832], [0.8035193681716919, 1.1075878143310547, 1.9465858936309814, -3.022599220275879, 0.8855026960372925, 0.6209810376167297, 2.6916885375976562, -0.40931326150894165, -0.7286598682403564, 0.261883020401001, -1.3990834951400757, -1.3640228509902954, 0.6135121583938599, -1.6320226192474365, -1.0834375619888306, 1.2185560464859009, 0.616369366645813, -1.2684286832809448, 1.4197355508804321, 0.09837165474891663, -0.3274315595626831, 0.16301009058952332, -0.07891504466533661, 1.179919719696045, 0.5907275676727295, -0.8738268613815308, -0.5202476382255554, 0.7822255492210388, -0.6043688654899597, -0.9650158882141113, -2.3792288303375244, 0.8469629287719727, 0.008675879798829556, -0.20740407705307007, -0.8806806802749634, -0.5542364120483398, -0.0361345037817955, -1.1340765953063965, 0.09976045042276382, 1.7121368646621704, -0.5213478803634644, 0.0158638097345829, 1.5823760032653809, 0.10097140073776245, 1.5719095468521118, 1.0225367546081543, 0.9305958151817322, -0.16413745284080505, -0.4539240300655365, 1.333112120628357, 0.14839796721935272, -0.10384184122085571, -2.1861491203308105, -0.13131318986415863, 2.5089852809906006, 0.6596968770027161, 1.8052024841308594, 0.26568523049354553, -1.3633335828781128, 0.18430054187774658, 1.817602515220642, 0.2254113405942917, -0.2435978353023529, -0.1254524439573288], [-1.1273407936096191, -0.30678290128707886, 0.18408356606960297, 1.007245421409607, 0.8182203769683838, -1.0563197135925293, -1.8846975564956665, -0.30578288435935974, -0.7182192206382751, 0.8816930651664734, -0.9771134853363037, 0.6500548720359802, -0.3910331726074219, 1.180495023727417, 0.09764722734689713, -1.9364246129989624, -2.326242208480835, 0.33993229269981384, -0.9981257319450378, 0.9973939657211304, -0.020869823172688484, 1.8069126605987549, -0.01322134118527174, -0.7462223768234253, -2.008134126663208, 0.5002263188362122, -0.21392115950584412, -0.31688588857650757, 2.70031476020813, 0.7976053953170776, -0.9404081106185913, 0.6705132126808167, -1.549302577972412, -0.6488956809043884, 0.8580150604248047, -1.3166965246200562, -0.002242449903860688, 0.06336915493011475, 0.17389526963233948, -0.9803891777992249, -1.4209308624267578, -0.015031237155199051, 0.6358557939529419, -1.0020148754119873, 0.8084143996238708, 1.1043095588684082, 0.32442060112953186, 1.0261847972869873, 0.9863293766975403, 1.216808557510376, 0.27726149559020996, 0.30499300360679626, 0.3611016869544983, 1.2193549871444702, -0.49719586968421936, 1.5071804523468018, -0.9183217883110046, 1.9850945472717285, 0.6710376143455505, 2.741076946258545, 0.2953648865222931, 0.7170491814613342, -1.5355044603347778, -0.24160020053386688], [0.23035173118114471, 0.4512726664543152, 0.7002689838409424, 0.18588511645793915, -0.020422454923391342, 0.986422598361969, -1.4825314283370972, -0.4149706959724426, 0.7296224236488342, -0.04177474230527878, 1.2389990091323853, -0.6432457566261292, -0.3435799777507782, 0.6666651368141174, 1.8195815086364746, -1.2647656202316284, -1.6835395097732544, -1.1772680282592773, -1.2185548543930054, 0.29450681805610657, 0.9712257385253906, 0.01290073711425066, 0.2720201015472412, 0.6628457307815552, 0.41368475556373596, 0.32981154322624207, 0.270978718996048, 1.1760356426239014, -1.0972440242767334, -0.4595486521720886, 0.3020695149898529, 2.744908332824707, -0.8269578814506531, 0.8197419047355652, -0.7017862200737, -0.12611179053783417, -0.7437007427215576, 1.9193602800369263, 0.42006969451904297, -0.459930419921875, 1.0233185291290283, -0.6631523370742798, 0.31333446502685547, -1.2996395826339722, -0.26468712091445923, 0.38684210181236267, 0.384397953748703, -0.8066387176513672, -0.7274262309074402, -1.7075073719024658, -2.084909677505493, -0.19645905494689941, 1.1618103981018066, 0.5110016465187073, 0.6180379390716553, 0.04809848964214325, 0.012796456925570965, -0.0036893223877996206, 1.7159843444824219, -0.29239246249198914, 1.2473583221435547, -1.3711719512939453, -1.0330928564071655, -0.030937839299440384], [-1.4534504413604736, 1.5877515077590942, 0.28032374382019043, 0.46985962986946106, -0.40859249234199524, 0.5461751222610474, -0.5864737033843994, 1.4033607244491577, 1.3743858337402344, -0.5971161723136902, -0.7790312767028809, -1.4193953275680542, -1.368478536605835, 1.3592506647109985, 0.8901622295379639, 1.6870664358139038, 0.6417639851570129, 1.0144795179367065, 0.8079535961151123, -0.34841135144233704, 0.34505653381347656, 1.0906347036361694, 0.29581889510154724, -0.808639407157898, -0.8305936455726624, 3.1173787117004395, -0.7110452055931091, -0.5067533254623413, 0.22049134969711304, 0.07538223266601562, 1.1806684732437134, -0.2449997514486313, 0.8252784609794617, 0.2478552609682083, 0.25034207105636597, -0.01222191657871008, -0.7047547698020935, -0.348072350025177, -0.9887223243713379, -0.27914971113204956, 0.2795493006706238, 0.38812455534935, -0.9812436103820801, -0.5982367992401123, -0.18218640983104706, 1.467629313468933, 0.3585329055786133, 0.5158430337905884, 1.3789113759994507, 0.6407270431518555, -0.3610503077507019, 1.8214436769485474, 0.02097543701529503, 1.3352537155151367, 0.5559730529785156, 0.7857832312583923, 1.4477472305297852, -0.2973117232322693, -1.2787977457046509, 0.4580390751361847, 0.08680824935436249, -0.12419930845499039, 0.14738775789737701, -1.4336656332015991], [0.11309107393026352, -0.1667010337114334, 0.4109669029712677, 0.01962805911898613, -0.252989798784256, 0.7525773644447327, 1.1807031631469727, 0.009176774881780148, -0.7648316025733948, -0.5947780013084412, 1.282982349395752, -1.5955275297164917, 0.09632227569818497, 0.3107394874095917, 0.49746307730674744, -1.9795383214950562, 0.22296489775180817, -2.180830240249634, -0.5805161595344543, -1.0273971557617188, -2.083286762237549, 1.620255947113037, -0.480985552072525, -1.7515766620635986, -1.2753428220748901, -0.31034770607948303, 0.5307537913322449, -0.14811673760414124, -1.3603079319000244, 1.409977912902832, 0.2054949551820755, 1.5031423568725586, 1.2679322957992554, 0.21525603532791138, 1.2690597772598267, -0.1754191517829895, 0.9741461277008057, -1.6886873245239258, -1.8811297416687012, -1.1591875553131104, 0.45454639196395874, 0.07814662903547287, -0.7875313758850098, -0.6560848355293274, -0.6241730451583862, 0.0701427310705185, 1.0197868347167969, 0.9252178072929382, -0.246258944272995, -0.0731838271021843, 0.5370272994041443, -1.1194283962249756, -0.037257060408592224, 0.5378785133361816, 0.4163036048412323, -0.48437991738319397, -0.8481769561767578, -0.9332056641578674, 1.3815529346466064, 0.004185743164271116, -1.2492139339447021, -2.0324342250823975, -0.0012230048887431622, 1.484907627105713], [-0.2852284014225006, -0.8980448246002197, 0.7887653112411499, -0.333074152469635, -0.15907470881938934, 0.2706158459186554, -0.3367537260055542, 0.6610546708106995, 0.803743839263916, -0.10491452366113663, -0.5684873461723328, 0.2959232032299042, -2.5477612018585205, -0.43735185265541077, -0.574709951877594, 0.7080899477005005, -1.0851657390594482, 1.0531055927276611, -0.1559278517961502, 0.3917694091796875, 1.2776176929473877, 0.13758280873298645, -0.6245824694633484, 0.2689867615699768, 0.2914106845855713, 1.9139188528060913, -0.0689375251531601, 0.9147412180900574, 1.3252241611480713, 1.1022758483886719, -0.22185564041137695, -1.1573714017868042, 1.0608969926834106, 1.7141567468643188, 0.6813824772834778, 1.2061262130737305, 0.19358403980731964, 0.890209436416626, 1.3827964067459106, 0.07748039066791534, 1.2969330549240112, 0.04575445130467415, -0.830608069896698, 0.469716876745224, -0.07496832311153412, 0.5358691215515137, 0.6505110859870911, -0.08989182859659195, 0.4967024028301239, -0.3564531207084656, 0.8994153141975403, -0.8239049911499023, -1.280012845993042, 1.1010522842407227, -1.8915107250213623, -0.19339871406555176, -0.1414763629436493, 0.8939318060874939, -0.1832791119813919, -0.1838146150112152, -1.7774192094802856, 0.9990590214729309, -1.022274374961853, -0.005682750139385462], [-0.777435302734375, 1.1304495334625244, -0.9250909686088562, -1.3490660190582275, 1.298081874847412, -0.6556217670440674, 0.22845375537872314, 0.4188864231109619, -1.5086030960083008, 1.0030704736709595, -1.0337074995040894, 0.061122260987758636, 0.4351845383644104, -0.08686766773462296, -0.8916633725166321, 0.929705023765564, -0.8886168599128723, -1.322123646736145, 1.171439528465271, 1.230987787246704, -1.878366470336914, -0.22098511457443237, -0.012426040135324001, -2.8408007621765137, -0.5758945941925049, 1.137115240097046, -1.0194538831710815, 1.3009381294250488, 1.973767876625061, -0.1630055457353592, -0.743348240852356, 0.05804387107491493, -1.4529953002929688, 1.2814292907714844, 0.4957069754600525, -1.427046298980713, -0.8022620677947998, -0.3241294324398041, 1.0888526439666748, 0.836169958114624, -0.6527667045593262, 0.17185263335704803, -0.44802284240722656, 0.4788268506526947, -0.27778080105781555, 2.1177985668182373, 0.46831974387168884, -0.26938575506210327, -1.0667262077331543, -1.4131758213043213, 0.7295089960098267, 0.8277548551559448, 1.5849480628967285, -1.0354958772659302, 0.7566903829574585, -1.2410223484039307, 0.6890734434127808, 0.573789119720459, 0.23780126869678497, -0.3094487488269806, -0.486259788274765, -0.37309759855270386, -1.9478912353515625, 0.67911696434021], [-1.1434104442596436, 0.15981535613536835, -0.05379559099674225, 0.725807249546051, 0.07691055536270142, 0.02950834482908249, 0.30519717931747437, 1.0108662843704224, -0.3782815635204315, -0.8681812882423401, 0.0511961355805397, -0.08230289816856384, 0.7649745941162109, 0.4328940808773041, 0.19973082840442657, 1.1731125116348267, 0.34558340907096863, 0.675385057926178, -0.3522491157054901, 1.5677682161331177, 0.7143169641494751, -0.8778802156448364, -0.8327956199645996, 1.8438302278518677, 0.2975374460220337, 1.8098818063735962, 2.439117670059204, 0.2193886637687683, 0.38346973061561584, 0.8977211713790894, -0.8163151144981384, -1.7417490482330322, 1.9581966400146484, -0.7247838973999023, -0.04142562672495842, 1.5978004932403564, 0.17470817267894745, 1.0384231805801392, 0.4251341223716736, -1.183004379272461, -0.6662361025810242, 0.19472649693489075, -0.8675304055213928, 1.0816898345947266, 0.4670279026031494, -1.485260248184204, 0.5791328549385071, -0.4230351448059082, -0.22250095009803772, 0.8114640712738037, -1.786007285118103, -0.8571941256523132, 0.35326680541038513, -1.4082790613174438, 1.3620105981826782, -3.2235703468322754, 1.1367980241775513, 0.795490026473999, -0.6096446514129639, 0.11581864207983017, -0.70713210105896, 1.930295705795288, 0.01676533743739128, 0.5297750234603882]]], [[[0.3240806460380554, -1.3678534030914307, -1.9216625690460205, 0.19571027159690857, 1.455841302871704, 0.6762587428092957, -0.1477973461151123, 0.4661216139793396, -0.3326398432254791, 1.3233586549758911, 0.6402652859687805, 0.3462657034397125, 2.1564290523529053, -0.5545221567153931, -0.9038958549499512, -0.21247628331184387, -0.5039451718330383, -0.5528506636619568, 0.6496098041534424, 0.562119722366333, -1.7146941423416138, -0.6664971113204956, -0.5114247798919678, -1.850893259048462, 0.6288632154464722, 0.12625189125537872, 0.480479896068573, -0.6603986024856567, -0.6047845482826233, 0.23425008356571198, 0.062325816601514816, -0.47108694911003113, 0.4740225076675415, -0.7133165001869202, 0.9183318614959717, 0.39484554529190063, -0.6064939498901367, -1.3082178831100464, 0.27151912450790405, 1.0833890438079834, 0.01165634673088789, 1.4111902713775635, 0.7336990833282471, 1.2809360027313232, -1.5086115598678589, -0.6716793775558472, 0.09154467284679413, 0.22495202720165253, -0.1563691645860672, -0.4259496331214905, -2.1936912536621094, 1.1164088249206543, 0.751190721988678, -0.49839040637016296, 0.022532103583216667, -0.6546835899353027, 0.9935736656188965, -0.25065097212791443, 1.1508915424346924, -0.5228586196899414, -0.9375754594802856, -1.6657934188842773, 0.416054904460907, -0.648723304271698], [0.5414428114891052, 1.0652812719345093, 0.307271271944046, 1.4959256649017334, -0.4146920144557953, -0.8145109415054321, 0.27764415740966797, -0.16281543672084808, 1.5854729413986206, -1.7585898637771606, -1.1065566539764404, 1.2899729013442993, -0.803180992603302, -0.24482229351997375, -0.5156071186065674, -0.06362397968769073, 2.0720388889312744, 1.0703520774841309, -0.007531690411269665, 1.9056308269500732, -0.9686286449432373, 0.176248237490654, 1.9178740978240967, -1.2776687145233154, -0.9462248682975769, -0.955971896648407, -0.37081918120384216, 0.7791385054588318, 1.5112498998641968, 1.1446962356567383, 1.322918176651001, 0.3622460663318634, -0.541554868221283, -0.9122899174690247, 0.002183526288717985, -0.913764476776123, -1.2283377647399902, 0.542465329170227, -0.8754155039787292, -0.3201415538787842, 2.069988489151001, 0.48145902156829834, 0.5056977868080139, -0.2505156695842743, -0.9197723865509033, 0.5628900527954102, -0.4241677224636078, -0.18308478593826294, -0.6374444961547852, 0.3763900101184845, 0.22274883091449738, -2.2196829319000244, -0.9814381003379822, -0.2099340707063675, 1.8388869762420654, -0.9947884678840637, 0.3375232219696045, 1.0879923105239868, 1.0771023035049438, -0.5437743067741394, 0.8907426595687866, 0.23739676177501678, 0.05376880243420601, -0.5204508304595947], [-1.803023099899292, -0.5493133664131165, -1.5938389301300049, 0.8351545333862305, 0.38948968052864075, 1.7228944301605225, -1.534836769104004, -1.6342179775238037, -0.2599266767501831, 0.13811154663562775, 0.8086139559745789, -1.226036548614502, -1.8970955610275269, -1.085283875465393, 3.300605297088623, -1.1031277179718018, -0.7220099568367004, 0.7452561259269714, 0.49810877442359924, -0.5398253798484802, -3.1994268894195557, 0.5183766484260559, -0.6529751420021057, 1.2164244651794434, -0.03439180552959442, 0.47351735830307007, -2.005758285522461, 0.007817280478775501, -0.15269735455513, -0.5296470522880554, 0.3557344377040863, 0.5215749740600586, -0.49737805128097534, 0.5819966793060303, 1.4654425382614136, 0.4993889629840851, -0.1841312050819397, 0.27192771434783936, -0.22531485557556152, 0.25572875142097473, 0.7854230403900146, -0.35654959082603455, 0.6618443727493286, 1.660173773765564, -0.22056207060813904, -0.7647355198860168, -0.4060629606246948, 0.6630030870437622, 0.3919092118740082, -0.5968066453933716, -2.6187360286712646, 0.4890161454677582, 0.43832284212112427, 0.41469696164131165, -0.7040902972221375, -2.2411065101623535, -0.38688182830810547, -0.772457480430603, 0.5731558799743652, 1.5451154708862305, 0.8699582815170288, -0.28885456919670105, 0.8150743246078491, -0.5147525668144226], [0.774875819683075, 0.10919222980737686, -0.4634404480457306, 0.4234868586063385, 0.19661743938922882, 1.438188910484314, -0.019295869395136833, -0.08849328756332397, 1.4673798084259033, 0.43405938148498535, 0.30220746994018555, 2.1054508686065674, -2.3599815368652344, -1.3833439350128174, -0.4919191598892212, 1.4510550498962402, 1.6868363618850708, 0.48839855194091797, -0.43337148427963257, -1.1848162412643433, 0.7935181856155396, 0.6275580525398254, -0.32427501678466797, 1.303299903869629, 1.9426275491714478, -0.6267013549804688, 0.5900862812995911, -1.0156712532043457, 0.5118036270141602, -1.0017659664154053, 0.3267207443714142, 2.1813621520996094, -0.3241310119628906, 1.7359906435012817, 1.6164398193359375, 1.4702837467193604, -0.14000746607780457, -1.18061363697052, 0.2884925603866577, 0.87519770860672, -0.33240559697151184, -0.6647538542747498, -1.0363606214523315, 3.231355905532837, -1.8582448959350586, 0.9991834759712219, -0.3084733784198761, 1.367713451385498, 1.4582114219665527, -0.8536527156829834, 0.1040748804807663, -0.15155692398548126, 0.736409068107605, 0.04138306900858879, 0.9767021536827087, 0.668829083442688, -0.4192309081554413, 1.2396997213363647, 0.6507455110549927, 0.6967126131057739, -0.25227710604667664, 0.8942233920097351, -0.23650552332401276, 1.4801119565963745], [0.5928309559822083, -2.1389002799987793, 0.7977235317230225, 0.7035781741142273, 0.2202211618423462, 0.987251877784729, -1.325478434562683, 0.5145424008369446, -0.5983438491821289, 0.378528356552124, -0.5617437362670898, -1.1301753520965576, -0.1791311353445053, -1.5344429016113281, 0.18851712346076965, 0.8742617964744568, 1.4413152933120728, 0.5508870482444763, -0.32710781693458557, -0.0483885183930397, 0.02827717736363411, 1.3887914419174194, -0.8073453903198242, -0.38403618335723877, -1.9343458414077759, 1.3846020698547363, -1.0681082010269165, -1.1452676057815552, -1.821010708808899, 0.3776572048664093, -0.15825296938419342, -0.2503286898136139, -1.2847799062728882, -0.44067302346229553, 1.2569974660873413, -1.4418058395385742, 0.20821894705295563, 1.4275099039077759, -1.0566538572311401, 0.7179573774337769, 0.7354019284248352, -0.7170832753181458, -0.019570140168070793, 0.728198230266571, 0.9398220181465149, -0.5282438397407532, -1.4046063423156738, 0.2257322371006012, 0.22266478836536407, 0.7346832752227783, 1.4495301246643066, 1.185120940208435, 0.9539763927459717, -0.22919951379299164, 1.3462656736373901, 1.2469595670700073, -0.2644408643245697, -0.39962276816368103, -1.2755473852157593, -0.3912849724292755, 0.10604032129049301, -0.13747794926166534, 0.692025899887085, 0.14151883125305176], [-0.7692358493804932, 1.3322162628173828, -1.5929110050201416, 1.1265217065811157, 0.16440390050411224, 0.37914565205574036, -0.561843752861023, 0.23158057034015656, 1.716468334197998, -0.5230008959770203, 0.4197006821632385, 0.23364906013011932, -2.9572997093200684, -0.47725361585617065, 0.40757590532302856, -0.5801024436950684, -0.09796924144029617, -1.1183990240097046, 1.7162610292434692, 0.39009061455726624, -0.10636631399393082, -0.6637086868286133, 1.0307412147521973, 0.22040659189224243, 0.8949521780014038, 0.6844693422317505, -1.272091269493103, 0.5709240436553955, -2.308544158935547, -2.6406824588775635, -0.23886987566947937, 1.748420000076294, 0.5827145576477051, 0.5868804454803467, -0.831283450126648, -0.13493883609771729, 0.5361659526824951, -2.089913845062256, -0.7653740048408508, -0.4763548672199249, 0.9014745354652405, 1.229602575302124, -0.5839805006980896, 1.5176352262496948, 0.16222669184207916, -0.06325722485780716, 1.3406933546066284, -0.7634558081626892, -0.6397113800048828, -1.2340511083602905, -0.7788592576980591, -2.221921920776367, 2.488284111022949, 0.7014798521995544, 1.9129794836044312, -1.5365393161773682, -1.831802248954773, 1.4670580625534058, -0.2779097259044647, -0.34215447306632996, -0.48074328899383545, -1.3485223054885864, -0.734042227268219, 2.119088888168335], [-1.0937628746032715, 1.46792733669281, 0.38787829875946045, 0.8205649852752686, 1.162490725517273, -0.9827112555503845, 0.8757071495056152, 0.4536117911338806, 2.746666431427002, 0.4512075185775757, 1.0482177734375, 0.2391623854637146, 0.40618932247161865, 0.31035521626472473, 0.009234639815986156, 1.0952003002166748, 0.35260236263275146, 0.5628849267959595, 0.937782883644104, -0.6196615695953369, -0.24927493929862976, 0.357412725687027, 0.5408771634101868, 1.8080708980560303, -2.0749099254608154, -0.749607503414154, -0.9511321187019348, 0.0850684717297554, -0.23895052075386047, -1.709924340248108, -0.9410361647605896, -0.9363610148429871, -1.7693899869918823, -1.0548179149627686, -0.6413883566856384, 1.0338715314865112, 0.08378145843744278, -0.5365338921546936, -0.7737843990325928, -0.24408479034900665, 1.1843680143356323, 1.2808606624603271, -0.8704200983047485, -0.37676674127578735, 0.1332108974456787, 0.2899605929851532, -1.0702589750289917, 0.5765387415885925, 0.8096770644187927, 0.8899059295654297, 1.8787288665771484, 0.9865936040878296, -0.1559108942747116, -1.419926643371582, -0.06226211413741112, -0.6617076992988586, 1.104094386100769, -0.08852441608905792, -1.2490501403808594, -0.5025092363357544, 1.4913378953933716, -0.6656333208084106, 0.1882135272026062, 1.0533355474472046], [0.34556710720062256, -1.3727366924285889, -1.4131646156311035, -0.8346819877624512, 0.44925224781036377, 0.2663756012916565, -0.7568382024765015, 0.011371023021638393, 0.23882469534873962, -1.6815943717956543, -1.2659319639205933, 0.06686481833457947, 1.1025456190109253, -0.19889360666275024, 0.18772520124912262, 1.4884920120239258, -0.8106886744499207, -0.02987673692405224, -0.2881856858730316, 1.8541910648345947, -0.5197080373764038, -0.8500446081161499, 0.984063982963562, 2.941011905670166, 0.9557345509529114, 1.3183172941207886, 1.338739037513733, 0.5710428357124329, -0.13785943388938904, 1.8556818962097168, 0.5640795230865479, -1.0602965354919434, -0.42747873067855835, 0.07436669617891312, -0.7656161785125732, 0.01915420964360237, -0.35148945450782776, -0.3571849465370178, -1.4221549034118652, 1.5407733917236328, -1.0759063959121704, -1.267290472984314, 0.6938391923904419, -0.163191556930542, -0.09963271021842957, 1.8423821926116943, -0.7013712525367737, -1.269592523574829, -0.8409980535507202, -0.6261940002441406, 0.2970099449157715, 0.5634037852287292, 1.1823716163635254, -0.40109992027282715, 0.517025887966156, -0.9443305134773254, -0.4681321680545807, 0.1562928706407547, -1.1318333148956299, 1.7320783138275146, -0.4181094765663147, -0.896774411201477, -0.9102271199226379, 0.24188244342803955], [1.717306137084961, -1.3495492935180664, 0.15143030881881714, -0.8831983208656311, 0.834039032459259, -0.5362573862075806, -2.2259268760681152, -1.1682049036026, -0.3086549937725067, -0.33190372586250305, 1.5183475017547607, 0.41749194264411926, -1.051293969154358, 0.7881765365600586, -0.14137689769268036, 2.0652332305908203, -0.28867900371551514, -0.6490789651870728, 0.29939067363739014, 1.7889533042907715, -0.39325469732284546, 1.5432764291763306, 0.07910846173763275, 0.2537718117237091, 1.3100544214248657, 1.119065523147583, 0.6159331202507019, 1.9124252796173096, -0.4499320685863495, -2.454457998275757, 0.7417799830436707, 0.231427863240242, -0.19995218515396118, 1.225999355316162, -2.2055985927581787, -0.30375638604164124, 1.310563325881958, 0.3109147846698761, 1.2712477445602417, 0.2703321874141693, -1.2198766469955444, -1.2181570529937744, 0.1788962334394455, -1.3295496702194214, 1.6056102514266968, -0.8961048722267151, -1.82364022731781, -0.2922292947769165, 0.504667341709137, -0.11795013397932053, -0.22850355505943298, -0.49848753213882446, -1.829942226409912, 0.11415518075227737, 0.767385721206665, -1.0313938856124878, 0.4766293168067932, -1.7563821077346802, 0.16852223873138428, -1.0866023302078247, 0.8074703216552734, -1.4986063241958618, -1.2568796873092651, -0.37685462832450867], [-0.08961267024278641, -0.35765644907951355, 0.6653513312339783, -0.05698070302605629, -0.35984256863594055, 1.6284034252166748, 0.9158811569213867, 0.20308683812618256, 0.327973872423172, -1.8808190822601318, -1.306904673576355, 2.120358943939209, -1.5506370067596436, 0.9441961646080017, 0.39116573333740234, -0.437399685382843, 2.5407304763793945, -0.2983216941356659, 1.4734892845153809, -0.6977433562278748, 0.2806962728500366, 0.3104807436466217, 0.19660913944244385, 1.1373932361602783, 0.269223153591156, -1.3233532905578613, 0.17706887423992157, -1.7846089601516724, 1.3841500282287598, 0.964046835899353, 0.8791561126708984, -0.24478214979171753, 0.4822591245174408, -2.262791395187378, -0.6630811095237732, 0.11351140588521957, -2.3652944564819336, 0.13565871119499207, -0.4509366750717163, 0.26246944069862366, -0.633801281452179, 1.3326425552368164, -0.10827410966157913, 1.1771965026855469, 0.8529519438743591, -1.2928738594055176, -2.1866183280944824, -0.38439106941223145, 0.27305397391319275, 0.31291475892066956, -0.519102156162262, 2.1992383003234863, -0.1357472687959671, 1.495552659034729, -1.75140380859375, 0.4746882915496826, -0.16530963778495789, 1.0645830631256104, 2.5297794342041016, -0.6903075575828552, -0.5940001010894775, -0.12490882724523544, -0.31560778617858887, -0.024828974157571793], [0.040798645466566086, -0.587444007396698, 0.007899089716374874, -0.23677779734134674, 1.3452060222625732, -0.6073068380355835, 0.8240179419517517, 0.5986742973327637, 0.16685166954994202, 1.1304454803466797, -1.9383450746536255, 0.4103485345840454, -0.7540855407714844, -2.3641984462738037, -0.39639726281166077, 0.3113541603088379, -2.0502066612243652, 0.26978227496147156, -1.0393439531326294, -0.8783811330795288, 1.5180894136428833, 1.1839470863342285, 0.6623528599739075, 0.39367109537124634, -0.9402511119842529, 1.8601704835891724, 1.109928011894226, 0.7128311395645142, 1.45442533493042, 0.21132813394069672, 0.05869413912296295, 0.7973194122314453, 0.8306521773338318, 1.6861491203308105, -0.38253119587898254, 1.8551812171936035, 0.10898423939943314, 1.0866895914077759, 0.37622225284576416, -1.8155654668807983, -0.13791286945343018, -0.14933717250823975, -0.3754732310771942, 2.5752830505371094, -0.3785961866378784, 0.5970938205718994, 0.7128015756607056, -0.4201318323612213, -1.430572748184204, -0.9412431716918945, 1.5892817974090576, -0.7443103790283203, 0.3464356064796448, 0.3096601366996765, -1.1070961952209473, 0.832427442073822, 0.5256113409996033, -1.3402981758117676, -0.9707240462303162, -0.4504861533641815, 1.0859100818634033, -0.7349144816398621, -2.0300445556640625, 1.7827898263931274], [-1.0944197177886963, -0.6299670934677124, -0.6534979343414307, -2.067902088165283, 1.3160443305969238, 0.9990230202674866, 1.0820132493972778, 0.2403668910264969, 3.2732276916503906, -1.4326262474060059, 0.08576612174510956, -0.32789289951324463, -2.4104762077331543, -0.8524674773216248, -0.26041722297668457, -0.1670616716146469, -0.3967569172382355, -0.07948307693004608, 0.16059094667434692, -0.2841622829437256, 1.280636191368103, -0.08225618302822113, 1.3942691087722778, 1.501094102859497, 0.18753927946090698, -0.8707332611083984, 0.4392290711402893, 1.3750724792480469, -2.310857057571411, -0.30985191464424133, 2.475829601287842, -0.2092994749546051, -0.7502995729446411, -0.7794491052627563, -0.622065007686615, -0.5144359469413757, -1.3651959896087646, 0.13037057220935822, 0.013852369040250778, -1.431694507598877, 1.274221658706665, 1.11076021194458, -0.5641658902168274, -0.28695979714393616, 0.4127286672592163, 0.015705671161413193, 0.5900428891181946, 1.1439707279205322, 1.636017918586731, 1.0924015045166016, -0.1641114503145218, -0.32468101382255554, -1.5454367399215698, 0.15364602208137512, 1.716671347618103, 0.6499969363212585, -0.6306609511375427, 0.37544941902160645, -0.39376747608184814, 0.10394161194562912, -0.6198724508285522, 1.7189702987670898, -2.2070703506469727, -1.0571423768997192], [0.652655839920044, -2.2411932945251465, -0.4433800280094147, -1.314754605293274, 0.3448668420314789, 0.009701893664896488, -0.09811045974493027, -0.00970652885735035, -0.23702266812324524, 1.98464035987854, -0.35759878158569336, -1.0788922309875488, 0.061641838401556015, -1.590362310409546, -1.123458981513977, 1.0264973640441895, 0.17820684611797333, -0.11718665063381195, 1.1401007175445557, -0.9621736407279968, -1.9316285848617554, 1.063501000404358, -0.27034300565719604, 2.513941764831543, -1.6334583759307861, 0.0718780979514122, -0.5751168131828308, 0.2411363124847412, -0.08716582506895065, -0.36485517024993896, 1.592492938041687, 1.1292383670806885, 0.40056121349334717, 0.6978684663772583, 0.36867234110832214, -1.8088046312332153, -0.8574612736701965, 0.36830082535743713, 0.21920114755630493, 1.808619499206543, -0.7493268251419067, -0.130567267537117, 0.06377211958169937, 0.2709585130214691, 1.3073303699493408, -0.8643958568572998, 1.0922484397888184, 0.43319085240364075, -1.488219976425171, -1.3377563953399658, -0.08540556579828262, 1.8580971956253052, 0.40956032276153564, 0.6514831781387329, 0.3587650954723358, -1.221747636795044, -0.6507357358932495, -1.6597981452941895, 0.8998413681983948, -0.42286476492881775, 1.3308764696121216, -0.07909727096557617, -1.501686453819275, -0.4217556416988373], [1.0586249828338623, -0.3201506435871124, 0.08488515764474869, -1.5912894010543823, -0.38501161336898804, 0.7834437489509583, -0.09614534676074982, 0.23204046487808228, 0.29744574427604675, -0.8104091286659241, 0.07524097710847855, -0.07016508281230927, -0.7050802707672119, 1.842074990272522, 1.6934869289398193, 1.407400369644165, -0.04928077384829521, -1.5695983171463013, 0.8214635252952576, -0.670816957950592, 0.8401554226875305, -0.374175101518631, -0.6954014897346497, -1.9354413747787476, 0.34189870953559875, 0.6357929706573486, 0.6959586143493652, 0.9965394139289856, -1.6866213083267212, 0.19383804500102997, 0.9462235569953918, 1.1370705366134644, -1.6307276487350464, 0.32061833143234253, 1.1263102293014526, 0.7430835962295532, -0.8926247358322144, 1.0083364248275757, 0.331466943025589, 1.539768099784851, -0.6651175022125244, 0.20702241361141205, 0.06969541311264038, 0.7518766522407532, -1.419697880744934, -0.04124514013528824, -1.3127243518829346, -0.1893422156572342, -0.562842607498169, -0.029891226440668106, 0.8500314950942993, 0.40397512912750244, 0.5295793414115906, 0.4711354374885559, -0.10630005598068237, -0.6290395259857178, 0.359324187040329, -1.782119870185852, 0.2458002120256424, -0.5170716047286987, -0.7352774143218994, 0.08773908764123917, 0.10938441008329391, -1.110041618347168], [-0.013981220312416553, 0.5594531297683716, -0.7203802466392517, 0.7293131947517395, -1.4648712873458862, -0.8783168792724609, 0.7245556712150574, -0.2732691168785095, -0.5666722655296326, 0.508558988571167, 0.7557640075683594, -0.053451597690582275, 0.37059924006462097, 0.06824751198291779, -0.5704014897346497, 0.26177820563316345, -0.2726665139198303, -2.108954906463623, -0.5292848944664001, 0.5081583261489868, 0.03297705203294754, -0.038254667073488235, -0.6426332592964172, -1.5597003698349, -1.0235408544540405, -1.3948842287063599, -1.5805702209472656, 0.449943870306015, -0.5095123052597046, -1.2747927904129028, 0.4535273611545563, 0.7786860466003418, -1.5304663181304932, 0.6446101665496826, -1.300795555114746, 2.562025308609009, 0.31455302238464355, -2.0598866939544678, 1.1535439491271973, -1.0212448835372925, 0.4341508448123932, 1.725103497505188, 0.9578354358673096, -1.6462304592132568, 0.11101148277521133, -0.6522006392478943, 0.2298366129398346, -0.34214988350868225, -0.14784203469753265, 1.5492324829101562, -0.644245982170105, 1.2221477031707764, -0.9852613806724548, -0.23082248866558075, -1.5296506881713867, 1.1576136350631714, 0.07138579338788986, -1.3866899013519287, 0.10306810587644577, -0.41614899039268494, -0.47738492488861084, -0.5634847283363342, 0.6296229958534241, -0.6759562492370605], [1.0127488374710083, 0.6117656230926514, 0.9512954354286194, 1.10902738571167, 0.4446694552898407, -0.7094317674636841, 0.8943852782249451, -1.4269027709960938, -0.6764688491821289, 1.1463491916656494, -1.8082952499389648, -0.8859174847602844, -1.3372513055801392, 0.3736979067325592, -1.02860689163208, 0.19449077546596527, 0.043949492275714874, 1.979201078414917, 2.0958123207092285, -0.8327865600585938, -0.6717298030853271, -0.05900539085268974, -0.3766714334487915, -0.3415672481060028, -0.15892665088176727, 0.9098086953163147, 1.372442603111267, -1.4148451089859009, 0.5944443941116333, 0.14774231612682343, -0.9890608787536621, -0.05647042393684387, -0.5188241600990295, 0.10929341614246368, 0.10399053990840912, -1.7016546726226807, -0.5163238644599915, 0.23603498935699463, -0.6285950541496277, -3.1091034412384033, 1.3137089014053345, -0.1568969339132309, 0.6279906034469604, -0.6560253500938416, -0.8922040462493896, 0.9882558584213257, 0.10492151975631714, -0.486226350069046, 0.3620564341545105, -0.3193877935409546, 0.2488717883825302, 0.8613300323486328, 0.04190696403384209, -0.4379061758518219, 0.6884281635284424, 1.204564094543457, 0.25200197100639343, 0.03607378154993057, 0.04703017324209213, -0.6341204643249512, 0.6808624267578125, -0.39703401923179626, -0.3545634150505066, 0.4031873345375061]], [[1.4778506755828857, 1.1717076301574707, -1.062908411026001, 1.439521312713623, 0.8025926947593689, 0.1514747589826584, 0.9817519187927246, 0.2966782748699188, 1.922666072845459, -0.7173696756362915, 1.4971003532409668, -0.41782450675964355, -0.2691618502140045, 0.8883659839630127, -1.9553759098052979, 0.9094483256340027, 1.319114327430725, -1.680073618888855, -0.633281946182251, 1.060203194618225, -0.9616311192512512, -1.3203486204147339, 1.8076236248016357, -0.3892544209957123, 1.6459420919418335, 1.0479435920715332, 0.391729474067688, -0.7843952178955078, 1.1159218549728394, 0.5206010341644287, -0.5596349239349365, -0.7297086119651794, -0.33412590622901917, -1.1359909772872925, -0.4860326647758484, 0.4842219650745392, -0.8894041180610657, -0.35507723689079285, -0.30706682801246643, -0.13724498450756073, -0.6799649000167847, 0.4193349778652191, 1.1986234188079834, -1.2843053340911865, -0.5419999361038208, -0.1517971009016037, 1.1768007278442383, -0.13478809595108032, -0.522971510887146, 0.8836696743965149, -0.1663135439157486, -1.0703206062316895, 0.25000908970832825, 0.10676278918981552, -0.6321485638618469, -1.6834800243377686, -0.5440505146980286, 0.3968663215637207, 0.3381110429763794, -1.5034687519073486, -1.1678626537322998, 0.9026063680648804, -1.8426244258880615, 0.8732308149337769], [0.6527636647224426, 2.015495538711548, -1.6676045656204224, 0.8358821272850037, -0.6080018281936646, 0.027867209166288376, -1.0483587980270386, 0.9192641973495483, 0.5542119145393372, -1.1629598140716553, -1.4636321067810059, 0.3239675760269165, -0.47687599062919617, 0.19949114322662354, 0.18514060974121094, 0.35669150948524475, 0.0713285431265831, -0.5130950212478638, 1.0061553716659546, 0.7355831265449524, -0.035393279045820236, 0.38582512736320496, -0.008950408548116684, -0.10535839200019836, -0.4893485903739929, 1.3763456344604492, -0.5269622206687927, 1.3603503704071045, 0.6695204973220825, 0.5828642249107361, 0.7016631364822388, 0.44255688786506653, -1.2608070373535156, -1.492821455001831, -0.7636751532554626, -0.5978853702545166, -0.04890973120927811, -0.17302407324314117, 0.3164033889770508, -1.7196252346038818, 0.09780445694923401, -0.7836015820503235, -0.7513781189918518, -0.7647578120231628, -0.08814771473407745, -1.0017905235290527, 0.6183966994285583, 0.6447989344596863, -0.7979119420051575, -0.7958347797393799, -0.11332419514656067, 1.109632134437561, 0.5300443768501282, -0.7856763601303101, -0.2061791568994522, -1.3263742923736572, -0.5754131078720093, -1.1959589719772339, 0.4208774268627167, -0.36765044927597046, -0.17918722331523895, -1.7079375982284546, -0.5871677398681641, -0.522705614566803], [0.9729191660881042, -1.8086309432983398, -1.0716532468795776, -1.0886861085891724, 0.29471346735954285, -0.7221387624740601, 1.069154143333435, 1.445449948310852, -2.0357542037963867, 1.177430272102356, 0.6065773367881775, 0.7102747559547424, 0.44544118642807007, -0.3412761390209198, -0.5641161799430847, -0.6523860692977905, -0.9445432424545288, 0.22162765264511108, -0.13771748542785645, 0.12488869577646255, 0.09333427250385284, -1.726348638534546, -0.1268308460712433, -0.8091046214103699, -0.9525368213653564, -0.1212654709815979, 0.7907422780990601, -0.2882816791534424, 0.22042728960514069, 1.5082272291183472, 0.6470072865486145, 0.5331809520721436, -0.7266383171081543, 0.8471091985702515, -0.34290778636932373, 1.5095937252044678, 1.9556955099105835, -0.6744992733001709, 0.3035876750946045, 0.2052280753850937, 1.5523861646652222, -1.2408734560012817, 1.2250313758850098, 1.1440564393997192, 0.16911372542381287, 0.6299594640731812, -0.015245816670358181, 0.40197306871414185, -0.0847088173031807, -1.29597008228302, -0.47954854369163513, -0.4434421956539154, 0.6289181709289551, -0.3469647467136383, 0.5434751510620117, 1.1056432723999023, -0.30568599700927734, 0.5861474871635437, -1.2443137168884277, 0.20009282231330872, 1.0718218088150024, 1.014975905418396, 3.5673112869262695, -0.4409671127796173], [0.18436837196350098, 0.13930298388004303, -0.5929490923881531, 0.028835270553827286, -1.1736884117126465, -0.2304847538471222, 3.0203986167907715, -0.6822523474693298, 0.9831365346908569, 1.3868460655212402, 1.5015201568603516, 0.7088115811347961, 0.3626127243041992, 0.9326211810112, 1.060015082359314, -1.435847282409668, 1.5338211059570312, 0.1744007170200348, 1.0180249214172363, 1.0685348510742188, 0.4230239689350128, -1.1126775741577148, -1.6690762042999268, 0.9320728778839111, -0.6101217269897461, -1.4374793767929077, -0.4998711347579956, -0.002834590617567301, 1.563658356666565, -1.0722678899765015, 0.46185317635536194, 0.15708063542842865, -0.4446231722831726, -1.014357566833496, 1.4231094121932983, -0.2042001336812973, 0.2998610734939575, -0.2489088475704193, 1.818052053451538, 0.31657183170318604, -0.4377683401107788, 0.9046235084533691, 0.15963315963745117, -0.9867634177207947, -1.4866127967834473, -0.6034660935401917, -0.550509512424469, -1.5842849016189575, -2.1086671352386475, 0.08413279801607132, 0.003395203733816743, 0.5348941683769226, -1.2621228694915771, -0.09992027282714844, -0.0009984277421608567, -0.4402512311935425, -0.9596166610717773, -0.07679780572652817, -0.6163137555122375, -0.7154316902160645, -0.4595522880554199, -1.0566318035125732, -1.1085840463638306, -0.2633673846721649], [-0.10762066394090652, 0.6040247082710266, -0.3608371615409851, -0.9232916831970215, -1.2750123739242554, 0.7855157852172852, -0.6493817567825317, -1.1817185878753662, 1.3302538394927979, 0.920102596282959, -0.6697805523872375, -0.19037850201129913, -0.9396185874938965, -0.22762344777584076, 1.2986669540405273, -0.2039039582014084, 0.9434242248535156, 1.2223377227783203, -1.909683346748352, -0.7994086742401123, 1.2624633312225342, -0.6573380827903748, -0.6673133969306946, -0.7998536825180054, 0.8196184635162354, -0.17181019484996796, 2.3786094188690186, 0.2351534515619278, -2.117511034011841, -1.295159935951233, 0.09008654206991196, 1.8315507173538208, 0.9193454384803772, -0.7152693271636963, -0.6145361065864563, -1.6496227979660034, 0.9659966230392456, -1.6817933320999146, 0.7210935354232788, 0.8792908787727356, 0.3623609244823456, 1.1138193607330322, 2.1475753784179688, -0.8094921708106995, 1.7383432388305664, -2.136404275894165, -0.4001534879207611, -0.21634535491466522, -0.1001284196972847, 1.1321702003479004, -0.23220698535442352, -0.9391445517539978, -0.9851633310317993, -0.4003034830093384, 0.5093330144882202, -1.1112287044525146, -0.7496314644813538, -0.6930408477783203, 0.4449767470359802, 0.27159377932548523, -1.166816234588623, -1.252226710319519, -0.7757284045219421, 0.265610933303833], [1.239011526107788, 0.08439398556947708, -0.733564019203186, 2.068077564239502, 0.7658389210700989, -2.429597854614258, 0.6259574294090271, 1.11574125289917, 1.0360976457595825, -0.5827099680900574, -1.6387712955474854, 0.041770175099372864, 0.6230417490005493, 0.5107347369194031, 0.30974385142326355, -0.7572271823883057, 0.06357128173112869, -0.0908389613032341, -0.551916241645813, 1.7818981409072876, -0.34286391735076904, -0.3243288993835449, 1.8203895092010498, -0.4510249197483063, -0.4770224988460541, -0.1829509437084198, -1.3457223176956177, 0.5456275343894958, -0.4376218914985657, 0.14485102891921997, 1.0282936096191406, -0.6867288947105408, -0.5724254250526428, -1.6912389993667603, 0.6811192035675049, -0.5750874280929565, 0.7118764519691467, -0.19357949495315552, -0.6427901387214661, -0.9549718499183655, 1.0946266651153564, 0.5498635768890381, 0.9548963308334351, -0.15797169506549835, -1.4800033569335938, -0.21443970501422882, 0.5532279014587402, 0.6352826952934265, 0.8350695371627808, 1.027267575263977, 0.9047111868858337, 1.6210627555847168, -1.7838870286941528, 0.6620467305183411, 0.6275163888931274, 0.8135183453559875, 0.11769786477088928, -0.7375169992446899, 0.2566777765750885, 0.7859528660774231, 0.11987647414207458, 1.0928168296813965, 0.9399109482765198, -2.0839767456054688], [2.3498482704162598, -1.0708492994308472, 2.4623608589172363, 0.7926816940307617, -0.5858049988746643, 0.3179114758968353, -0.06710135191679001, 0.6199281215667725, -1.3996895551681519, -0.3038055896759033, -2.373401641845703, -0.0018137304577976465, -0.5057306289672852, -1.1171354055404663, -1.0261698961257935, -0.18624024093151093, -0.6032435297966003, -0.17400169372558594, -0.5202615857124329, -0.581221342086792, -0.6659215092658997, 1.0639845132827759, -0.3995739817619324, -0.0237455815076828, 0.7303004860877991, 1.3135446310043335, -1.4243388175964355, -0.3537812829017639, -0.965740442276001, 0.17820216715335846, 0.16999439895153046, -1.679962158203125, -1.0156183242797852, -0.8059147000312805, 0.9065230488777161, -2.142859935760498, 0.5250865817070007, 0.02191033400595188, -0.6009505391120911, 0.15073111653327942, -0.7178676128387451, 1.9166771173477173, 0.08365476876497269, -0.4903925359249115, 1.1148492097854614, 0.07285487651824951, -0.8576450347900391, -0.2296055257320404, 0.405775785446167, 0.08062738925218582, 0.2561505436897278, 0.26827287673950195, 0.7135244607925415, -0.05450987443327904, 1.8493956327438354, 0.5918864011764526, 1.2361071109771729, 0.04546704888343811, -0.6481087803840637, -0.027397265657782555, 0.19679543375968933, -0.5326281189918518, -2.902712345123291, -0.551166832447052], [1.15830397605896, 1.5810022354125977, 1.5065174102783203, -0.3532458543777466, -0.3678855299949646, 1.7277625799179077, 0.7343124747276306, -1.1649571657180786, -1.0895551443099976, -0.902155876159668, -0.5989304184913635, -2.403777837753296, -0.7736782431602478, -0.62723308801651, 0.65675288438797, -0.31350794434547424, -1.7329713106155396, 0.3420681655406952, 0.9381769299507141, 1.7533916234970093, 0.9790100455284119, 0.887135922908783, 1.7484426498413086, -1.2241500616073608, 1.8061933517456055, 1.8824130296707153, -0.7167029976844788, 1.161973476409912, -1.3483363389968872, 2.9494147300720215, 0.8988279700279236, 0.031326454132795334, -0.8932703733444214, 0.15789411962032318, 0.7961908578872681, 0.07977962493896484, 1.1627459526062012, 0.3850899338722229, -0.48946869373321533, 1.1001945734024048, 3.189073085784912, -2.0696680545806885, 0.11438465863466263, 1.0663965940475464, -0.5741991400718689, -1.0296965837478638, -0.7000496983528137, 0.19054685533046722, -1.0445725917816162, -1.7761727571487427, -1.4074822664260864, 0.2319781631231308, -0.17965900897979736, 0.9651010036468506, 0.1207309141755104, -1.1784987449645996, -0.7218870520591736, -0.8613566756248474, 0.8150973916053772, -0.10899592936038971, -0.2095891684293747, 0.2920805811882019, 0.5511004328727722, -0.7352827191352844], [-1.4503347873687744, -0.11593370884656906, -0.47743460536003113, 0.3524051904678345, -1.3727442026138306, 1.393254280090332, -0.11905152350664139, 1.2917627096176147, 0.5210446715354919, -0.11773208528757095, -0.934425413608551, -0.16670095920562744, -0.8871771097183228, 1.6970772743225098, -0.8113023042678833, 0.04613278806209564, -0.7830929160118103, -0.6320576667785645, 0.16719764471054077, 2.114938974380493, -1.1465184688568115, 0.08030100166797638, 1.0949546098709106, 0.01656132936477661, -0.47267940640449524, 2.8966751098632812, 0.37298583984375, -0.5921692252159119, -1.6615408658981323, -0.9433864951133728, -0.5785303711891174, 0.9783948659896851, 0.035062529146671295, -0.8032646179199219, 1.7297933101654053, 1.1550370454788208, -0.41155683994293213, 0.9132372736930847, 0.04969555512070656, -0.5175139307975769, -1.0658416748046875, -0.20912989974021912, 0.11002882570028305, 0.3161357641220093, 0.17875659465789795, 1.0579993724822998, 1.5154297351837158, -0.02087102085351944, 0.6771761775016785, 1.274680733680725, -0.24154174327850342, -0.7013731598854065, -2.0616202354431152, -0.24207167327404022, -2.1513290405273438, -1.0055196285247803, 0.41423743963241577, 0.8350659608840942, -1.3102165460586548, -2.1053590774536133, -1.380631446838379, 0.3243478834629059, 2.095951795578003, -0.11143551021814346], [-2.0513670444488525, -1.1758959293365479, 1.1898902654647827, 0.4166967272758484, 0.638815701007843, -0.14055192470550537, -0.08746081590652466, -2.0720248222351074, 0.912452220916748, 1.6349561214447021, -0.6527641415596008, -0.7882694602012634, -0.09787412732839584, -0.378527969121933, -1.284877061843872, -1.162109375, 0.23162078857421875, 1.1910109519958496, -0.6086152791976929, -1.5126032829284668, 0.4439091384410858, 1.1310937404632568, -0.7105218768119812, -1.1273638010025024, -0.5661898255348206, -0.2694997191429138, 1.5110726356506348, -1.0620620250701904, 0.3518851101398468, 0.45665988326072693, -0.44298484921455383, 1.7807590961456299, 0.6445503234863281, -0.6059103012084961, -0.28282439708709717, 0.050780102610588074, 1.6694740056991577, -0.315757691860199, 0.7461427450180054, 0.44474464654922485, -1.026462435722351, 0.3434520959854126, -1.0671406984329224, -0.4233205318450928, -0.14164400100708008, 0.6045663952827454, -0.11205337941646576, -0.42172592878341675, -0.5652124285697937, -0.45104923844337463, 0.17213821411132812, -1.3549357652664185, -0.7203165292739868, -0.020635245367884636, -2.486882448196411, 0.6467735171318054, 0.8566967248916626, -0.725279688835144, -0.07952059060335159, -0.09864263981580734, -0.4496613144874573, 0.13944891095161438, -1.1889351606369019, 0.7590968012809753], [-0.6861369609832764, 1.0561244487762451, 2.6870663166046143, 0.26473096013069153, 0.48823845386505127, 0.2995580732822418, -0.11562822014093399, 0.6959853768348694, 1.0416502952575684, -0.5440841913223267, 1.3410097360610962, -0.24565909802913666, 1.4944301843643188, -0.19581827521324158, -2.1069986820220947, -0.41552937030792236, -0.9402577877044678, -0.20668281614780426, 0.8323878049850464, -0.6934300661087036, -0.6725265979766846, 0.15019501745700836, -1.4208316802978516, -0.47103556990623474, 1.3006439208984375, 1.4739469289779663, -0.5705240964889526, -0.3819601237773895, 0.7485178709030151, 1.2367621660232544, -0.3563151955604553, -0.8675207495689392, -0.7140162587165833, 1.2010722160339355, 0.5514052510261536, -0.7877522706985474, 1.657217025756836, -0.28888264298439026, -0.7833958864212036, 1.0739519596099854, 1.0733839273452759, -0.658380925655365, -0.5101917386054993, 0.6535712480545044, -0.2902759313583374, 1.2732518911361694, 0.8794219493865967, -0.4909162223339081, -2.785942792892456, 0.10056933015584946, -0.01538501214236021, 2.225454330444336, -1.002367377281189, -0.10403374582529068, 1.6493291854858398, -0.1668483316898346, 0.9911093711853027, 0.2756485641002655, -0.01437193900346756, 0.6910043954849243, -0.3619421422481537, -0.8401368260383606, 0.5256587862968445, -0.28461506962776184], [0.4774847626686096, 1.3869210481643677, 0.8919420838356018, -1.107468605041504, -0.8812538981437683, -1.3305654525756836, -0.5001844763755798, -0.6995905637741089, 0.5013925433158875, 1.2758833169937134, 0.46279457211494446, -2.0223660469055176, -0.23175691068172455, 0.07605810463428497, 1.3366448879241943, 0.4811779260635376, 0.8144302368164062, 0.2928929626941681, -1.4625734090805054, 0.5484941005706787, 1.0686147212982178, -1.0800366401672363, 1.7852076292037964, 2.3322341442108154, -0.8176872730255127, -0.5964502096176147, -1.0260601043701172, 0.3039773106575012, -0.8102144598960876, 1.7806837558746338, -0.5206454396247864, 0.006084630265831947, 0.7532508969306946, -0.3666476309299469, 1.661636471748352, 0.004683622159063816, 0.24243110418319702, 0.7920036911964417, -1.2625850439071655, 0.21334950625896454, -0.5892475843429565, 0.2808832824230194, -1.071326732635498, 1.4275027513504028, 0.6683521270751953, -0.32268622517585754, -0.08973655849695206, -1.6086349487304688, 0.48136091232299805, -1.1775418519973755, 0.873238742351532, -0.6808737516403198, 1.6913580894470215, -0.265873521566391, -0.5241625308990479, 1.1472299098968506, -0.12409181892871857, 2.026481866836548, 2.0486998558044434, -0.5572580099105835, 1.2261829376220703, 0.6659159064292908, 0.21805201470851898, -1.8022713661193848], [-0.05654536932706833, -0.4776655435562134, 1.1926982402801514, 1.2784969806671143, -1.0840094089508057, -1.352274775505066, 0.24642325937747955, -0.38440269231796265, 0.9618211388587952, -0.50386643409729, 1.3858751058578491, 1.7485994100570679, -1.2116349935531616, -0.9311841726303101, 0.16196748614311218, -0.2306334525346756, 1.954773187637329, -0.0922708585858345, -0.18976104259490967, -1.5163140296936035, 0.5690574645996094, -0.9476891756057739, -0.6061000227928162, -1.9566380977630615, -0.2070157378911972, 0.15937048196792603, 0.1928853690624237, -1.5225390195846558, 0.1829889863729477, 0.3792528510093689, 0.7313146591186523, -1.4871175289154053, 0.23890957236289978, 0.2612496018409729, -0.0760100781917572, 0.34789663553237915, -1.0448111295700073, -0.8278177976608276, 2.286282539367676, 1.1180987358093262, 0.8371188640594482, 0.4227409362792969, -0.3331139087677002, -0.721198558807373, -1.3401923179626465, -0.16838869452476501, -1.6816905736923218, -0.00589874479919672, -0.8009464144706726, -0.38837796449661255, -0.5258849263191223, 1.8006924390792847, 1.943821668624878, -1.1254684925079346, 2.2226641178131104, 0.9667279720306396, 0.632936954498291, 1.1696771383285522, -2.4694859981536865, 0.3038582503795624, 1.8215856552124023, -0.397116482257843, 1.3514695167541504, -0.5575533509254456], [-0.9451057314872742, 0.5276089906692505, -0.281904399394989, -0.5971977114677429, -0.1940467655658722, 0.19336630403995514, 0.8282409906387329, 0.0034860463347285986, 0.19435609877109528, -0.08308684080839157, -0.6756108403205872, -0.7915363311767578, -0.34897059202194214, -0.6120470762252808, -0.4492850601673126, 0.6536174416542053, 0.13391342759132385, 0.11103737354278564, -0.8472356200218201, -0.40043285489082336, -0.3318954408168793, -0.644652247428894, 0.752005934715271, 1.8611152172088623, 1.065205693244934, -0.9258942604064941, -0.3567332923412323, 0.6750712990760803, -0.25203773379325867, 0.124990314245224, -1.2066665887832642, -1.2169902324676514, 0.5921534895896912, -0.6818658113479614, 0.6822362542152405, 0.9620978832244873, -0.7281033992767334, -0.8037620782852173, 1.6192330121994019, -0.106594517827034, 0.44247353076934814, 0.42737066745758057, -0.11280351132154465, 0.12017311900854111, 0.7475338578224182, -0.13255898654460907, -1.7780025005340576, -0.37565070390701294, -0.5137608051300049, -0.42468252778053284, 1.0375975370407104, -1.616410255432129, 0.6744170188903809, -1.2442947626113892, -0.5670765042304993, 0.5068130493164062, -0.8604033589363098, 1.297218680381775, 0.9931066036224365, -0.3269566297531128, -0.34139567613601685, -0.4370616376399994, -2.5845959186553955, -0.7899416089057922], [-0.7233834862709045, 0.10579467564821243, 0.6232095956802368, -3.34254789352417, -0.8290130496025085, 0.667599618434906, 0.6080718636512756, 1.4520139694213867, -0.5872042179107666, -1.276552677154541, -0.04261567443609238, 0.5261742472648621, -1.069409966468811, -0.9480657577514648, -1.2489149570465088, 0.5761538743972778, -1.2343143224716187, 0.07732219994068146, -0.0430561788380146, -0.28202009201049805, 0.3582209050655365, -0.025060798972845078, -1.3215785026550293, -0.9385651350021362, 0.32126176357269287, 1.7163664102554321, 0.7268438935279846, 0.0422331802546978, 0.22044673562049866, -0.8781000375747681, 1.420742392539978, 1.1155004501342773, -2.2221121788024902, 0.6948548555374146, -1.7874395847320557, -0.04900744557380676, 1.2307541370391846, -0.8020151853561401, 0.07729672640562057, -0.22445333003997803, -3.2846643924713135, 2.083717107772827, 0.5839570760726929, -2.6031744480133057, 1.4761407375335693, -0.6968451738357544, -0.5695924162864685, -0.5524478554725647, 0.9880194067955017, 0.49623537063598633, -0.9830886125564575, -0.09211330115795135, -1.1616291999816895, 1.1547589302062988, 0.27897292375564575, 0.9197271466255188, -1.3264442682266235, -0.010691859759390354, 1.8354469537734985, -0.5726884603500366, -0.11663331091403961, -0.5527477860450745, -0.7236055731773376, 0.11061165481805801], [-0.21891574561595917, -0.8760361075401306, -0.8337336778640747, 0.5058481097221375, 1.4370992183685303, 0.36911630630493164, 0.48022300004959106, -0.6535972356796265, -0.34790775179862976, -0.8374881148338318, 0.02206178940832615, 0.013182888738811016, 1.2302244901657104, 1.3357030153274536, -0.4501102864742279, 0.7637640833854675, -0.15581750869750977, -1.1720982789993286, -0.5930302739143372, 0.3229505121707916, 0.2740674912929535, 0.6931359171867371, -0.7698983550071716, -0.22606179118156433, -1.8208484649658203, 0.9152842164039612, 3.4374983310699463, 0.3508489429950714, -2.1545486450195312, 0.02427651360630989, -0.0293339304625988, 0.8218543529510498, 0.609031617641449, 1.5338897705078125, 1.2194589376449585, 0.4631563723087311, -0.6331737041473389, -0.6024254560470581, 0.19330397248268127, -0.42363524436950684, 1.1290391683578491, 0.7487088441848755, 0.8166717290878296, -2.481024742126465, -0.9183474183082581, 1.1916863918304443, 0.1680484414100647, 0.15365883708000183, 0.8641029000282288, -0.5934807658195496, -0.6296699047088623, -0.36915087699890137, -1.213889479637146, -0.24535802006721497, -2.0327625274658203, 1.4960826635360718, 1.768652081489563, 0.729274332523346, 0.20396143198013306, -1.9716213941574097, 2.104060649871826, 0.6950942277908325, -0.7469016909599304, -0.8515223860740662]], [[0.215719535946846, -0.7890348434448242, 0.45944616198539734, 0.9960584044456482, -1.2871137857437134, -1.545632243156433, -0.6743455529212952, 0.9968156814575195, 0.2716684341430664, 0.1493447870016098, -0.19335272908210754, 1.206682562828064, -0.9945492744445801, 1.3325273990631104, -0.7734037041664124, -0.421720951795578, 0.4113522469997406, 1.3482378721237183, 0.20328286290168762, -1.3184165954589844, 0.08138656616210938, -1.6370784044265747, 0.8903610110282898, 0.13461121916770935, -0.08006524294614792, 0.6031178832054138, -1.1619292497634888, -1.9120562076568604, -0.35646969079971313, -0.42201247811317444, -1.9919483661651611, 0.14886035025119781, 0.973504900932312, 1.5702497959136963, 0.6665960550308228, 0.011287190951406956, -2.704575538635254, -2.480792284011841, 0.9032522439956665, 1.4930813312530518, -0.7116197347640991, 0.11985185742378235, 0.605167806148529, 0.3636355400085449, 0.8009308576583862, -0.7459663152694702, -1.7445483207702637, -0.2033787965774536, -0.025012679398059845, -2.7039244174957275, -0.12161006033420563, 2.755110740661621, 0.8847886919975281, -0.24220021069049835, 0.6795443296432495, 0.656981348991394, 0.13499975204467773, -0.26785463094711304, -1.3704311847686768, -1.2016208171844482, 0.6364750862121582, -0.6345766186714172, -0.38839712738990784, 0.7103890180587769], [0.6898196935653687, -0.5182366967201233, -1.6068466901779175, -0.5054689645767212, -0.9463163018226624, 0.6288233399391174, -1.7542431354522705, 0.09206689894199371, 0.4256588816642761, -1.945345401763916, 0.29422688484191895, 0.24584831297397614, 1.6477679014205933, 0.542008638381958, 0.8491106629371643, 0.19968603551387787, 1.5556334257125854, -0.5951604247093201, 1.8135415315628052, 1.1261776685714722, -1.6873834133148193, -0.6011731028556824, 1.4067797660827637, 1.140290379524231, -0.2254898101091385, -0.3412075340747833, 0.7605263590812683, -1.5522981882095337, 0.026583373546600342, 1.407947063446045, -1.4376176595687866, 1.7180759906768799, 1.8576717376708984, -0.6231099963188171, -1.1760344505310059, -0.025834767147898674, 0.47607776522636414, 0.8948854804039001, -0.343863308429718, -1.1520988941192627, 1.2355684041976929, -0.7279402017593384, -1.1200921535491943, -0.8507999777793884, 0.6682566404342651, -0.5586385726928711, -0.5070435404777527, 0.981611967086792, -1.0465402603149414, -2.5902819633483887, 0.33759018778800964, 0.4032415449619293, 0.4132152497768402, -0.650031328201294, -1.4146579504013062, -0.47281256318092346, -1.1700899600982666, 0.9550498723983765, 0.31329578161239624, 1.9712265729904175, 0.7899218201637268, -0.22259646654129028, 1.4386425018310547, -0.8279028534889221], [1.196254849433899, 1.1264411211013794, -0.26473698019981384, -0.7888810634613037, 0.020853696390986443, -0.45334210991859436, 0.0996149480342865, 0.5201098322868347, 0.3421051800251007, 0.13095925748348236, 0.9654352068901062, 0.3531239330768585, 1.7259016036987305, -1.620312213897705, 0.2662351727485657, -0.5921801328659058, 1.2262197732925415, 0.12452042102813721, 1.2975975275039673, 1.6699188947677612, -0.1101493090391159, 1.1577295064926147, 0.9623712301254272, -0.6914200782775879, -0.564709484577179, 0.8470078706741333, -1.9299921989440918, -0.4031526744365692, -0.5591127276420593, -0.1526847779750824, 0.016906458884477615, 2.030733108520508, -0.40436941385269165, -1.2529168128967285, 0.05146360769867897, -0.9524176120758057, 0.9264519810676575, 0.8744187951087952, -1.4202691316604614, -1.3746707439422607, 0.30312424898147583, -0.48644018173217773, -0.05058625712990761, 1.5220937728881836, -0.10497400909662247, 0.4083549380302429, -1.2929778099060059, -1.444392204284668, -1.0376129150390625, -0.6254894137382507, 0.5712074041366577, 0.9332399368286133, 1.7052420377731323, -0.895646333694458, 0.8229856491088867, -0.781237781047821, -1.6984175443649292, 1.5907405614852905, 0.5636669397354126, -0.7735167145729065, -0.22781804203987122, 0.8484704494476318, 1.9563467502593994, 1.0391325950622559], [0.8286913633346558, -0.7347940802574158, -1.615257978439331, -0.006473924964666367, -1.3265957832336426, -0.4333399534225464, 1.3843841552734375, 0.282093346118927, -1.0594866275787354, -1.7616956233978271, -1.0979303121566772, 0.6807953119277954, -2.4529407024383545, -1.1823585033416748, 0.5488222241401672, -0.6138034462928772, -0.05295569449663162, -1.1356760263442993, 0.9453004598617554, 2.344595432281494, 0.5353612899780273, 1.532835602760315, -0.3714263439178467, 0.41627037525177, 0.04970235377550125, -1.7143573760986328, -0.09267795830965042, -0.037677861750125885, -0.4816981852054596, -0.3135668635368347, -1.823951244354248, 0.6384023427963257, -0.7880344986915588, 0.23204068839550018, -0.5069634914398193, 0.9938857555389404, 0.08191949129104614, 0.7019092440605164, -1.1911687850952148, 0.6750460863113403, 0.47283580899238586, -0.1456248015165329, -0.4782790243625641, 0.3552609980106354, 0.5156674981117249, -0.030132507905364037, 0.13364839553833008, -0.405962735414505, -0.68131023645401, -1.350484013557434, 1.042153000831604, 0.4190343916416168, 0.16358882188796997, -0.3169989585876465, -0.19954873621463776, -0.6331744194030762, 0.8837574124336243, 0.24794241786003113, 0.09455978125333786, -0.21314017474651337, -0.8213616609573364, 1.378279447555542, -0.14551204442977905, 0.9328216314315796], [-0.62176114320755, 0.1986643373966217, 1.1359578371047974, -1.3116897344589233, -0.8056994080543518, -0.9799292087554932, 0.3439631760120392, -1.7288790941238403, -0.4941812753677368, -1.477062702178955, -1.7083483934402466, 2.2749948501586914, -0.32272735238075256, 1.2523545026779175, 1.1838099956512451, 1.569666862487793, -0.139162078499794, -1.1326251029968262, 2.260035991668701, -1.6344363689422607, -1.8799909353256226, 0.6670931577682495, 0.9103142023086548, 0.8894047737121582, -0.29963478446006775, -1.3832966089248657, 0.2518208920955658, 1.0611765384674072, -0.08162710070610046, 1.4998586177825928, -0.07158855348825455, 0.9284087419509888, -0.7146397829055786, 0.4521181285381317, 1.512770414352417, 1.3887156248092651, 0.7150481939315796, -0.05897825211286545, -0.6227583289146423, -0.001390890683978796, 1.102766990661621, 0.8784970045089722, -1.0547735691070557, 1.3136777877807617, 1.5840703248977661, -0.6692838072776794, 0.515053391456604, -0.9876570701599121, 0.17001758515834808, -0.062388282269239426, 0.35507193207740784, -0.021324768662452698, -0.45788511633872986, 0.4718484878540039, -2.468313455581665, -1.1020554304122925, 0.013128926046192646, -0.09577500075101852, 0.5765439867973328, -0.9563531875610352, -1.4600740671157837, -0.03059164434671402, -0.5708193182945251, -2.019652843475342], [0.9232614636421204, 0.40748193860054016, -0.13205529749393463, 2.5841548442840576, -0.9323763251304626, 0.14126801490783691, 0.6972410678863525, -0.019493380561470985, -0.5074684619903564, 0.05821641534566879, 0.3699660897254944, 0.8616219162940979, -0.5769234299659729, -0.4470294117927551, -0.851676881313324, -0.5730496644973755, -0.5430657267570496, -0.6752426028251648, 1.8789315223693848, -0.601634681224823, -0.5553030371665955, 2.284723997116089, 0.839337170124054, -2.7908899784088135, -0.06614029407501221, -0.29702994227409363, -1.5700162649154663, -1.3409830331802368, 0.6956264972686768, 0.3860680162906647, 0.051935888826847076, -0.8930606245994568, 1.2549015283584595, 0.2207430899143219, 0.005342231132090092, 0.059168629348278046, -1.670541524887085, 0.3807331621646881, 1.4561846256256104, 0.34612277150154114, -0.507881224155426, 1.1356773376464844, -1.3353190422058105, -0.6403961777687073, 0.27470630407333374, 0.5331928133964539, 0.3766433894634247, -1.6821831464767456, 0.5091484189033508, 0.34131935238838196, -0.7811291813850403, 1.6300379037857056, -0.2739735543727875, -0.14292462170124054, 0.5675291419029236, 0.3372799754142761, 0.3434775173664093, -0.6091005802154541, 0.2305477261543274, -0.0070664179511368275, -1.1855071783065796, -1.5954086780548096, 2.2633488178253174, -1.1375826597213745], [-0.8352458477020264, 0.6895403861999512, 0.21924564242362976, -1.7912381887435913, 0.6076593995094299, 0.744586169719696, 0.005988870281726122, -0.4493226110935211, -0.035404354333877563, -0.9513779282569885, -1.239656925201416, 0.38977909088134766, 1.8019905090332031, 1.1065620183944702, 0.42001834511756897, 0.7838947176933289, 0.5054168105125427, 0.9687461256980896, 0.9649083018302917, -1.19186270236969, -0.7187884449958801, 1.5398460626602173, -0.85878586769104, -1.8298577070236206, 1.2785967588424683, 0.09199512004852295, 0.5409001708030701, 1.2822494506835938, -0.9324820637702942, 0.20751549303531647, 0.3195757567882538, -0.24133028090000153, -0.09103785455226898, -0.28334543108940125, 0.5140523314476013, -0.7393580675125122, -0.10627766698598862, -1.1299480199813843, 0.12714357674121857, 0.9476136565208435, -0.7605385184288025, -0.2523583471775055, -1.060828447341919, 1.8155885934829712, -0.019391732290387154, 0.4182718098163605, 2.131227731704712, -0.25396236777305603, -0.09531241655349731, -0.7413858771324158, -1.2621725797653198, 1.5981435775756836, 0.2761390805244446, 0.08378727734088898, 0.4385521709918976, -0.6730919480323792, 0.2765045166015625, 0.5380395650863647, 1.0719013214111328, -0.10434254258871078, -0.4210846722126007, 0.11686112731695175, 0.7484068274497986, 1.0358035564422607], [-0.3712756931781769, -0.6193749308586121, -1.4635848999023438, -0.5482596755027771, 0.20108459889888763, 0.5004163384437561, -1.1224216222763062, -1.762628436088562, -0.9395259022712708, 1.5066139698028564, 0.6433097124099731, -0.22144275903701782, 2.010270833969116, 0.21964344382286072, -0.27697181701660156, 0.249564066529274, 0.07494346052408218, -0.16062822937965393, 0.9187678098678589, -0.04942769184708595, 0.10794281214475632, 0.4221767485141754, 1.0131428241729736, -1.4308778047561646, -0.8599252104759216, -0.3672284185886383, -0.6743965148925781, -0.18707983195781708, -0.9847589135169983, -0.9671480059623718, -1.2636590003967285, 0.18556803464889526, 1.438539981842041, -0.9962128400802612, 1.4054213762283325, 1.2028369903564453, -1.9328242540359497, 0.23250187933444977, 0.3878405690193176, -2.1689293384552, 0.3181452453136444, -0.2554861605167389, 0.13337360322475433, 1.4374910593032837, 0.14067673683166504, 0.3930247724056244, -1.2400636672973633, 0.7485615611076355, 0.7086167931556702, 0.8094612956047058, 3.209430694580078, -0.08833437412977219, 0.8457263708114624, -0.15435971319675446, 0.6159102320671082, 2.2969675064086914, -0.28665605187416077, -1.0740448236465454, 1.201499342918396, -0.5396712422370911, -0.21388202905654907, 0.34531134366989136, -0.25507116317749023, 0.16606774926185608], [0.7740017771720886, 1.3758983612060547, -1.010311245918274, 2.244762659072876, 0.21196608245372772, -0.6677922606468201, -0.8562031388282776, 0.5832969546318054, 1.8201786279678345, -0.42257246375083923, -1.089469313621521, 1.4492202997207642, -1.0714870691299438, 0.5839068293571472, 0.7351388931274414, -0.016495466232299805, -1.3410420417785645, 0.756488561630249, 0.5641587972640991, -0.0029218322597444057, 1.004654049873352, -0.4106641113758087, -1.526393175125122, -2.2575302124023438, 0.23859737813472748, -0.5562986135482788, 0.6953704357147217, 0.3404626250267029, 0.15129968523979187, -0.6203209161758423, 0.063806913793087, 0.9556921720504761, -1.0382581949234009, 1.3428839445114136, -0.1775844693183899, 0.6758065819740295, -0.6917272806167603, 0.8570429682731628, -0.1043778732419014, -0.001517641358077526, 0.33716660737991333, 1.7465507984161377, 0.7136356234550476, 0.2239709049463272, -1.1539636850357056, 0.012101172469556332, -1.238045573234558, 0.3685716986656189, -0.36055371165275574, 0.031235797330737114, -0.8264997005462646, -0.01167932990938425, 0.2668876051902771, 1.6840014457702637, 0.5267770290374756, 0.7078707814216614, -0.9992595314979553, 0.8918015360832214, -0.5841625928878784, -0.9892972111701965, -0.3919677436351776, 0.23650150001049042, -0.44198405742645264, -0.5960873365402222], [0.5667828321456909, -0.871913492679596, -0.380439817905426, -0.587908923625946, -0.013651865534484386, 0.13961847126483917, -0.836245596408844, 0.9898120164871216, -2.8554301261901855, 0.33680614829063416, 1.2173455953598022, 1.5702567100524902, 0.5069186687469482, 0.5804793238639832, -0.4200607240200043, -0.40663957595825195, -1.0223637819290161, -0.568574070930481, -1.6494179964065552, -2.1037511825561523, 1.0410175323486328, 0.31300222873687744, 1.2333980798721313, -0.3030479848384857, -1.350504755973816, 0.2949436604976654, 1.1531990766525269, -2.011070966720581, -0.38047170639038086, 0.054904405027627945, 2.128750801086426, 0.22440873086452484, -1.2694746255874634, -1.209602952003479, -1.146852731704712, -0.9923461675643921, 0.5868125557899475, -0.23126612603664398, 0.056178655475378036, -1.5770328044891357, 2.105036735534668, -0.7164774537086487, -1.5560753345489502, -0.512919545173645, 0.5452418327331543, -0.4897894263267517, -0.2233441174030304, 2.0557546615600586, -1.0299097299575806, 0.6280494332313538, -2.668330669403076, 0.06683304160833359, -0.6971495747566223, 0.7145875096321106, -0.05761535465717316, -0.8951294422149658, -0.12770245969295502, -0.07688138633966446, -0.40133944153785706, -0.914107620716095, -1.3307371139526367, -0.9239300489425659, -0.5489113330841064, -1.0649186372756958], [0.4963129162788391, -1.561405897140503, -2.066798686981201, -1.492009162902832, -1.1996415853500366, -0.5139029622077942, 0.4937843084335327, -0.260924756526947, 0.7878021597862244, -0.11639469861984253, 0.07733634859323502, 1.099268913269043, 0.002393025206401944, 0.35443392395973206, -0.5562495589256287, -1.1504085063934326, -0.5187245011329651, 1.5052831172943115, 0.12850889563560486, -2.5081727504730225, -1.362428903579712, 0.3671036660671234, -0.7674247622489929, 0.6514360308647156, 0.23178640007972717, -0.4764327108860016, 1.6697980165481567, -0.34524449706077576, 2.0695712566375732, 0.3641725480556488, 0.5600192546844482, 0.25023820996284485, -1.0634558200836182, -0.35904693603515625, 0.3554201126098633, -1.0170890092849731, 0.9945668578147888, 0.08421322703361511, 0.09665525704622269, -0.09078460186719894, -2.5115504264831543, 0.3320245146751404, 0.8143190145492554, 2.077313184738159, 0.7856117486953735, -0.047735102474689484, 0.21725349128246307, -0.43049174547195435, -0.15159983932971954, -1.730840802192688, 0.09637484699487686, 1.4444059133529663, 0.2682484984397888, -1.032415509223938, -0.5011813044548035, 1.0629050731658936, 1.2705570459365845, -1.402072548866272, -0.9204955101013184, -0.16330304741859436, 0.6171742677688599, 0.14191630482673645, 0.2636215090751648, 0.9756128191947937], [0.6684443354606628, -1.3206429481506348, 0.49622589349746704, -0.43643462657928467, -0.3996821343898773, 0.07055272161960602, -0.14079627394676208, -0.14497819542884827, 0.9118967652320862, 2.021278142929077, -0.31508854031562805, 0.42562732100486755, -0.8233931660652161, -1.307288646697998, -1.3422399759292603, -1.473580002784729, -0.11141392588615417, -0.24878768622875214, 1.2382745742797852, 0.31413745880126953, 0.36372390389442444, -0.4757508933544159, -0.5638067722320557, 0.8231837153434753, 0.2501618564128876, 0.6020488739013672, 0.7098177075386047, -1.0660945177078247, -0.2369580715894699, 1.6201928853988647, 0.7105090022087097, 1.9236706495285034, -0.04942002147436142, 0.6037179827690125, 1.575691819190979, 3.098357915878296, 0.6821735501289368, -0.6302111744880676, 0.0034660242963582277, -0.6857143640518188, -0.4619493782520294, 0.08819479495286942, -0.5416510105133057, 0.41368597745895386, -1.6099778413772583, 0.16098198294639587, -0.8082388639450073, -0.3404647707939148, 1.4887067079544067, -0.5158957242965698, 0.13361112773418427, 0.6512170433998108, 1.0306516885757446, 0.9626544117927551, -0.37627479434013367, -0.67633455991745, 0.9881640076637268, 1.333283543586731, -1.118929386138916, 0.9854008555412292, -0.0168533343821764, 0.5753874778747559, -0.7428656220436096, 0.9817422032356262], [1.0569665431976318, -2.0024991035461426, 1.508378028869629, 0.28081706166267395, -1.0761548280715942, 0.9623749256134033, -1.0347895622253418, 0.788589358329773, 1.4554024934768677, -0.48890239000320435, -0.7082083821296692, -1.5783082246780396, -0.6206771731376648, 1.7796722650527954, -1.094179630279541, 1.4440689086914062, 0.8876802921295166, -0.37380409240722656, 0.6375541687011719, 0.103664830327034, 0.22190959751605988, 0.7557260990142822, 0.5829514265060425, -0.5211575627326965, 0.19264675676822662, 0.24666668474674225, -0.17053650319576263, -0.6662943363189697, -0.09407810866832733, -0.08009974658489227, 0.11214112490415573, -1.2102422714233398, -1.9722485542297363, 0.7821441292762756, 0.17914681136608124, -0.975715696811676, 0.8059186339378357, -0.3310781419277191, -0.8140372633934021, 1.292372226715088, 1.0878539085388184, -0.3116684854030609, 0.9851370453834534, 0.6020064353942871, -1.4095549583435059, -1.5808022022247314, 0.16800585389137268, -0.5817394256591797, -1.0385444164276123, -0.27645811438560486, 0.2438696026802063, 0.7480424046516418, -0.2147715836763382, -0.8249471187591553, 0.894482433795929, -1.2201793193817139, -1.230031967163086, -0.5986416339874268, -0.6802225708961487, 0.30633261799812317, -0.5755220055580139, 0.8812527060508728, -0.7378180623054504, -0.6719323396682739], [1.7047152519226074, -0.3520689010620117, -0.602418065071106, 0.29352790117263794, 1.1967328786849976, 1.2692457437515259, -0.8181480765342712, -1.7465485334396362, -1.1991254091262817, -0.46909138560295105, 1.0246649980545044, 1.289578914642334, -0.7801135182380676, -1.0157361030578613, 1.0710642337799072, -0.13918302953243256, -0.13066576421260834, 1.4851638078689575, 0.693018913269043, -1.3669573068618774, 0.3423025906085968, -0.006052742246538401, 0.9004168510437012, -1.4527206420898438, -0.22787746787071228, -0.8512318730354309, 0.04989797621965408, -1.026625394821167, 0.7068508267402649, -1.082442045211792, 1.0134449005126953, 1.0995088815689087, -0.14998850226402283, 1.3312879800796509, 1.338014841079712, -0.16784027218818665, -0.9498894810676575, -0.6024373173713684, -0.6238706707954407, 1.0353435277938843, 0.7977229356765747, 0.2786141633987427, 0.7325548529624939, 0.8908532857894897, -0.7932363152503967, -0.47650113701820374, -1.3007442951202393, 0.7603474259376526, 0.5230316519737244, 0.8778253197669983, 0.2476375848054886, 0.2967016398906708, -0.6265799403190613, -0.7391803860664368, 1.441675066947937, -2.0677647590637207, 1.2472718954086304, -1.22791588306427, -0.6884373426437378, 1.0220812559127808, -0.7787095904350281, -1.835768222808838, 0.6508118510246277, 0.8719595074653625], [0.4449690580368042, -1.3926411867141724, 0.27443167567253113, -0.06139953434467316, -0.31569039821624756, 0.2801777720451355, -1.024943470954895, 1.3174673318862915, -0.6301632523536682, 0.1548791378736496, 0.4129609167575836, -0.07001426070928574, 0.7945465445518494, 0.041081104427576065, 0.5517203211784363, -1.1782623529434204, 0.46982088685035706, 0.004740480333566666, -1.2917652130126953, -0.023715466260910034, -0.6925429701805115, -0.561538577079773, -1.506384015083313, -1.0710582733154297, -0.8297767043113708, -0.1290997415781021, 0.027186434715986252, -0.29170915484428406, -0.9366260170936584, 0.013616473414003849, 1.1745659112930298, 0.6639390587806702, 0.2909291386604309, 0.3353727459907532, -0.19486062228679657, 0.09311743825674057, 0.36810216307640076, 0.31239694356918335, 0.6737570762634277, 1.076369285583496, 1.1380754709243774, 0.524297833442688, -0.24860087037086487, -1.0942487716674805, -1.6498135328292847, 1.022222876548767, 0.6211788058280945, -0.07957213371992111, -1.0141019821166992, 0.29273414611816406, 1.387865424156189, 1.4318499565124512, 2.3910272121429443, 0.9516160488128662, 0.9512190222740173, 1.3018566370010376, -1.5704575777053833, 0.8587192893028259, -0.6908383369445801, 0.42054301500320435, -1.5439469814300537, -1.0254284143447876, -0.9492470026016235, -0.5431014895439148], [0.7233948111534119, -1.0671905279159546, 0.7102158665657043, 1.0388729572296143, -1.1404279470443726, 0.9865483045578003, 1.7776020765304565, -0.6226539015769958, 0.6478496789932251, 1.480932354927063, -0.5868710875511169, -0.6231276392936707, -0.1307341307401657, -0.13755415380001068, -0.9965152144432068, -0.3977276086807251, -0.9202225804328918, 0.557403564453125, 2.3761556148529053, 1.0811265707015991, -1.3868377208709717, 1.228020191192627, -0.935524046421051, 0.5965280532836914, -0.279348760843277, 1.4463459253311157, 0.06359265744686127, 0.12031606584787369, 0.015882888808846474, -0.11934179067611694, 0.3579094707965851, 1.8525935411453247, 1.6328771114349365, 0.78110271692276, 0.12531240284442902, -0.05019533634185791, 1.4214770793914795, -0.8233321309089661, -0.9639562368392944, -0.643222987651825, 0.3277381360530853, 0.34441307187080383, 0.4474184811115265, -0.21767795085906982, -0.7044922113418579, -2.061532497406006, 0.3853450119495392, -0.7282955050468445, 1.0543184280395508, -1.117812991142273, 0.12191388010978699, 0.5444864630699158, 0.357079416513443, 0.770431399345398, 0.9263312816619873, 0.3274388909339905, 0.9055979251861572, -0.01342124305665493, -0.0698675736784935, 2.6798934936523438, -0.050890929996967316, -0.25570249557495117, -1.833501935005188, -0.9767597913742065]], [[0.038281477987766266, -0.7735971212387085, 0.021758724004030228, -1.290818214416504, 1.223110318183899, 0.15264274179935455, -0.4423760771751404, -0.4224531948566437, -0.24355284869670868, 0.7376647591590881, -0.6746431589126587, 0.3806312680244446, -0.32882457971572876, -1.300124168395996, -0.4380318820476532, -0.0031836088746786118, 0.4442856013774872, 1.5505515336990356, -1.0369185209274292, -0.6713353991508484, 0.47545769810676575, -0.10785908252000809, -1.5473101139068604, 0.8536614179611206, -1.1364012956619263, -1.4680951833724976, -1.0881061553955078, -1.9614566564559937, -0.8376564383506775, 0.3315235376358032, 1.0519400835037231, -0.45519036054611206, -0.7931910753250122, 0.02703843265771866, -0.559280276298523, 2.75072979927063, 1.668297529220581, 0.5324532389640808, -0.2325395792722702, 0.5093146562576294, 0.8449515104293823, -0.15920484066009521, 0.8561148643493652, 2.8641560077667236, -2.0386078357696533, 0.33423271775245667, 0.9835999011993408, -0.7014506459236145, -1.048711895942688, 0.4508914053440094, -0.09566190093755722, -0.35191449522972107, -1.124457597732544, 1.5770654678344727, -0.2669842541217804, 0.8729398250579834, 0.05067698657512665, -1.2834104299545288, 0.01779438555240631, -1.8440262079238892, 0.5364727973937988, -0.0008468187297694385, -1.6429386138916016, 0.515324592590332], [1.4200340509414673, -0.8402597308158875, -1.991278052330017, -1.884737253189087, -0.6473025679588318, 1.5753830671310425, -1.2744665145874023, 1.8963240385055542, -1.2016324996948242, 0.11579283326864243, -1.6344393491744995, 0.7476978898048401, -0.29614976048469543, 1.233794927597046, 0.103936105966568, 0.4748380780220032, 0.7171542048454285, 0.459953248500824, 0.3818747103214264, -0.23145581781864166, -0.15102888643741608, -0.6347934603691101, 1.3002485036849976, -0.4673907160758972, -2.1454646587371826, 0.40371066331863403, 0.9520857930183411, -0.6246804594993591, -0.08968441188335419, -0.448149710893631, 0.6055550575256348, 1.4635823965072632, 0.1575136035680771, -0.07249808311462402, 1.0814580917358398, 0.5565301775932312, -0.4168778657913208, -0.022524289786815643, 1.6593611240386963, 0.24149754643440247, 0.4873451888561249, -0.951802670955658, 1.4929081201553345, -0.47988244891166687, 0.12101221829652786, 0.26706990599632263, -0.8128165006637573, -1.0834583044052124, 1.2434717416763306, 0.0760277584195137, 1.0035152435302734, 2.4543447494506836, -0.6869756579399109, 0.09037258476018906, 0.17637589573860168, -1.612082839012146, -0.7915107607841492, 2.0376086235046387, 0.07485342025756836, 1.1731663942337036, 2.0016653537750244, -0.3014466464519501, 1.6856789588928223, 1.1270383596420288], [-0.3084764778614044, 0.5642104148864746, 1.0287328958511353, -0.17441754043102264, 1.0070695877075195, 0.8592528104782104, -2.676353931427002, 0.3203377425670624, -0.6908507347106934, 0.7965518236160278, -0.03546055778861046, 0.9542853236198425, -0.3359668254852295, -1.1718419790267944, 1.7823712825775146, -0.3133690059185028, 1.029672622680664, -0.09886310249567032, -0.9996832609176636, -1.0366504192352295, 1.3897426128387451, 0.4461825489997864, 0.2595154047012329, -1.7561167478561401, 0.1688583642244339, -0.35086673498153687, 0.3951159119606018, -0.020496077835559845, -0.2854534983634949, 0.056494735181331635, -0.4297613203525543, 1.209861397743225, -0.5795184969902039, -0.4363517761230469, 1.286057710647583, 0.7864657044410706, 0.1788426637649536, 0.2059105932712555, 0.7374028563499451, -0.015345445834100246, -1.3074350357055664, 1.2314364910125732, -0.16548936069011688, -1.9827580451965332, -1.0510029792785645, -0.7575930953025818, -0.15954189002513885, 1.5111451148986816, -0.21218881011009216, -0.2237548679113388, -1.3096826076507568, 1.127375841140747, 0.043639831244945526, -1.0850536823272705, 0.7040722966194153, 1.5840582847595215, -0.3931346535682678, -0.17784075438976288, 1.0151631832122803, 2.2618961334228516, -1.3404477834701538, 1.450069785118103, -0.5893085598945618, -2.298924207687378], [0.8174931406974792, -1.7727570533752441, -0.8952125906944275, 2.0874528884887695, -0.8671852946281433, 1.1222206354141235, -1.2108792066574097, -0.3632550537586212, 0.430969774723053, -2.281891345977783, -0.6079671382904053, 1.8516777753829956, -0.7515299320220947, -0.26244089007377625, 0.018632451072335243, -0.6072034239768982, 2.066802978515625, -1.7029403448104858, 1.4047539234161377, 0.87850022315979, -0.017252996563911438, 1.2333168983459473, -1.5238844156265259, 0.11439212411642075, 1.006446361541748, 0.4349026083946228, -0.9236762523651123, 0.32567667961120605, 0.6640288233757019, -1.2346168756484985, -0.04779166728258133, -1.2451326847076416, -0.9613367915153503, -1.3764008283615112, 0.2765912711620331, -0.5604491233825684, -1.1097817420959473, 0.5066201090812683, -1.0108176469802856, -0.5151920318603516, -1.1983963251113892, -0.9168599843978882, 1.4421888589859009, -0.18622276186943054, -1.1528664827346802, 0.6586501598358154, -0.4964248239994049, -0.09020132571458817, 0.1589718610048294, 1.3513884544372559, 1.0484634637832642, -1.4813687801361084, 1.046642541885376, 1.538170576095581, -1.0175244808197021, -0.5012051463127136, 0.2403106391429901, 1.1787309646606445, -1.2767549753189087, 1.6043494939804077, 1.2013884782791138, 0.317741721868515, 1.9035661220550537, -1.0923784971237183], [1.996508002281189, 0.08641110360622406, 1.3108819723129272, -1.3447552919387817, 1.3745760917663574, 0.3623647093772888, -1.312438726425171, -0.22787296772003174, 2.150524616241455, -0.7434682250022888, 0.14053210616111755, 0.9357025623321533, 1.518667221069336, 0.08026202023029327, 0.3971741497516632, 0.22638583183288574, 0.8528579473495483, -1.4706530570983887, -0.8476631045341492, -0.9210489988327026, 1.3920220136642456, -1.2840622663497925, -0.8988012075424194, 1.0164191722869873, -1.9585981369018555, -1.0465610027313232, -0.6108725070953369, 1.359671950340271, 1.4255973100662231, 2.3126883506774902, -2.993704319000244, -1.3874874114990234, 0.6143301129341125, -1.9221243858337402, 0.8063139915466309, -1.5399023294448853, 1.0185858011245728, -0.5303502082824707, -1.2559502124786377, -0.32695263624191284, 2.153719902038574, 0.8433046936988831, 2.090036392211914, -1.0708286762237549, -0.07000871002674103, 0.7943392395973206, 1.1748913526535034, 0.06659334897994995, 0.21436768770217896, -0.42988070845603943, -0.07080940157175064, -0.5812516212463379, 1.0262020826339722, 0.40308472514152527, 0.006723609287291765, 0.49892091751098633, 0.7401888966560364, -0.460604190826416, -0.371256023645401, 0.22593453526496887, -1.7546731233596802, -0.5850371718406677, -0.6892315149307251, 0.764668345451355], [-1.2901188135147095, 0.020473003387451172, 0.3233400285243988, -1.1375141143798828, 1.9081567525863647, -2.2590575218200684, -0.9953539371490479, 2.2312698364257812, 2.819115400314331, 0.2919973134994507, 0.9786405563354492, -0.5304405093193054, 0.08263450115919113, -0.2482275515794754, 1.043709397315979, 0.5629456043243408, -0.6697129607200623, 0.7435029745101929, -1.7635571956634521, 1.9518953561782837, -1.3774383068084717, -0.9983581304550171, 0.7627129554748535, 0.11856646090745926, 0.8499701619148254, -0.23036816716194153, 2.1337826251983643, -2.285409450531006, 1.0890907049179077, 2.695960760116577, 1.114061951637268, 0.317135751247406, 0.7814971804618835, -0.04022393748164177, 0.9380069375038147, -0.1841518133878708, 0.5552043914794922, 0.03728075325489044, 0.07296070456504822, -0.779215931892395, -0.41164374351501465, -1.555912733078003, -0.4568081796169281, -0.12271203845739365, 0.3523021638393402, -0.1099400594830513, -0.49601149559020996, -0.17680180072784424, -0.46651965379714966, 0.14701934158802032, 1.0340675115585327, -0.47204703092575073, 0.3694378137588501, -0.2135612964630127, -0.09220227599143982, 1.1975001096725464, -1.8690019845962524, -0.2060755342245102, -1.0548170804977417, -1.0123869180679321, -0.7715350985527039, -0.1284681260585785, 0.6082091927528381, -0.9689562916755676], [-1.2474989891052246, 0.35806700587272644, 1.465334177017212, 1.117425560951233, 1.1232800483703613, -0.07473570108413696, 0.8215402364730835, 0.8016750812530518, -1.8521602153778076, 0.481738418340683, -1.1002578735351562, 1.6999613046646118, 0.05665784701704979, -0.500060498714447, 0.5419171452522278, 0.2536711096763611, 0.9344347715377808, 0.827714741230011, -1.2280470132827759, -0.6827393174171448, -0.15774306654930115, 0.605900764465332, 1.5551056861877441, -0.45965126156806946, 0.4063553214073181, 1.031361699104309, 0.7200781106948853, -0.6824256777763367, 0.04375452920794487, 0.7539891004562378, -0.05832012742757797, -0.02319895476102829, -0.46999701857566833, -1.0856297016143799, -0.41241219639778137, -0.9388788938522339, -0.3876267671585083, -0.2471262365579605, 1.6873306035995483, 1.015700101852417, -0.13052795827388763, 0.5827577710151672, -2.350677013397217, 0.15519529581069946, 1.38551664352417, -1.7359105348587036, -1.158949375152588, -0.661357045173645, -0.5808553695678711, 0.4778587818145752, 0.8121609687805176, -1.0576281547546387, -1.0206950902938843, 0.7189271450042725, 1.030380368232727, 1.090140461921692, 0.40127941966056824, -0.4614313542842865, -0.13636818528175354, -1.11601984500885, -1.1178617477416992, -1.2615872621536255, 0.024051157757639885, 0.20130743086338043], [-0.34913405776023865, 0.08163019269704819, 0.47446250915527344, 0.3263437747955322, -0.4781292676925659, -0.1672191619873047, 0.4021010398864746, 0.49983710050582886, 1.0581822395324707, 3.098681926727295, 0.7763373255729675, -0.6603796482086182, 0.8049245476722717, -0.1770569533109665, -0.7928357720375061, -1.2734228372573853, -0.35779470205307007, -0.75249844789505, 1.8444817066192627, -0.1810499131679535, 2.0381786823272705, -0.7165346145629883, -0.0654601901769638, -0.37204036116600037, -0.6677572727203369, 0.4558664858341217, 0.7374860644340515, -0.213368758559227, -0.03692160174250603, 0.048815689980983734, 0.1962134838104248, -0.5470411777496338, -0.8113362789154053, 0.6096752285957336, -1.1371744871139526, 0.8665317893028259, -0.3198443353176117, 0.1866666078567505, -0.34500497579574585, 1.375518798828125, 0.18081432580947876, 1.9067174196243286, -0.4787919819355011, -0.4695799946784973, 1.2053254842758179, 2.0853264331817627, -0.9223760962486267, -0.8396336436271667, 0.8346738815307617, -0.11057782173156738, 1.0043903589248657, 1.0492349863052368, -0.16617166996002197, -0.30670925974845886, 0.5078884959220886, 1.1778969764709473, 0.7835136651992798, -0.9055512547492981, 0.26973721385002136, 0.7856980562210083, 0.9078106880187988, 0.027340643107891083, 0.051227591931819916, -1.5183472633361816], [-0.6300894618034363, 1.83566415309906, -0.1455450803041458, -0.14522069692611694, 0.3694973289966583, -0.4246174991130829, -2.447916030883789, -2.07246732711792, -0.13274817168712616, 0.5626733303070068, -0.27668559551239014, 0.7453561425209045, -0.6462580561637878, 1.0655146837234497, 0.5842932462692261, -1.1563912630081177, -0.04400604963302612, 0.6392338275909424, 1.7789403200149536, -1.3228927850723267, -0.5049335360527039, 0.19721537828445435, -3.112128973007202, -0.3441047966480255, -0.06272953748703003, 0.09515140950679779, 1.5885111093521118, 1.2461236715316772, 0.5186995267868042, 1.0833635330200195, -0.43495434522628784, 0.3169543743133545, 0.549990177154541, -1.0901373624801636, 1.150537133216858, -0.1669478565454483, -1.9095181226730347, 1.1403839588165283, -0.8215040564537048, -1.4262313842773438, 0.15052339434623718, -0.03435678780078888, 1.2676067352294922, -0.9185501933097839, 1.5013359785079956, 1.0269508361816406, 0.4038267135620117, 0.5027269721031189, 0.9591103196144104, -0.7972054481506348, 0.7311956882476807, 0.5648959875106812, -1.566750407218933, -1.4464983940124512, 0.048598311841487885, 0.17966262996196747, -0.901223361492157, 1.25188410282135, 0.2904493510723114, 0.9778818488121033, -0.07078235596418381, -0.25239846110343933, 0.9736960530281067, 0.08135128021240234], [1.4764312505722046, 0.2859351933002472, 0.38838160037994385, -0.5348414778709412, 1.068085789680481, 1.8057951927185059, -0.23737139999866486, 3.4231441020965576, 0.15032753348350525, -0.01199411042034626, 0.6430261731147766, -0.3185162842273712, 0.4022747278213501, 1.0370306968688965, 0.44768333435058594, 0.11370252072811127, -0.6689943671226501, 0.6243605017662048, -0.14957737922668457, -0.8231247067451477, 1.47602379322052, 1.9025695323944092, 0.25762122869491577, 1.0016531944274902, 1.61436128616333, 0.8322001695632935, 0.37267300486564636, 0.2315828502178192, 1.3931628465652466, -0.9982144236564636, -0.8310883641242981, 0.6114102602005005, -0.6684849262237549, -1.2028636932373047, 0.13625185191631317, -0.03335382416844368, -0.7195682525634766, 0.7182945013046265, -0.1416747123003006, -0.6427590847015381, -0.3733343780040741, -1.369177222251892, 0.13933251798152924, -0.1835576593875885, 0.7436398863792419, -0.3926229476928711, 1.1520506143569946, 0.48766523599624634, 0.41144976019859314, -0.5018351674079895, 0.21435262262821198, -0.7475334405899048, -0.3346719443798065, 1.0008296966552734, 0.7018142342567444, -0.4643677771091461, 1.9028269052505493, 1.112383246421814, 0.7738712430000305, 0.7617173194885254, 1.2022106647491455, -0.0912233218550682, 0.5479734539985657, 0.7806429862976074], [-0.9923449754714966, -1.4749407768249512, 0.3072822093963623, -0.6114837527275085, 0.2798720896244049, -1.269793152809143, 0.6153466701507568, 1.5459872484207153, 0.12027372419834137, 1.4340656995773315, -0.07841476052999496, -0.42092207074165344, -0.49965983629226685, -2.070422410964966, -0.005637360271066427, 0.14242100715637207, 0.36555200815200806, -0.175174742937088, -0.099821537733078, -0.30373501777648926, -1.86897611618042, -0.841509997844696, 0.30759015679359436, -1.2255609035491943, 0.803866446018219, 1.8362350463867188, 1.8545925617218018, -0.14204616844654083, -0.4241408407688141, 1.2836079597473145, -0.5961849689483643, -0.6248025894165039, 1.4060499668121338, -1.238849401473999, -0.8854982852935791, 0.8212612867355347, -0.25308817625045776, -0.2350536435842514, 0.8225547671318054, 0.9437452554702759, -0.32607635855674744, 0.19290724396705627, 0.8223180770874023, 1.9302417039871216, 0.09638053178787231, -0.5904714465141296, -0.018125373870134354, 0.6289617419242859, 0.5269557237625122, -1.601881742477417, -0.22195270657539368, -0.8140097856521606, 1.1399801969528198, 0.39076387882232666, -0.37341493368148804, 2.134692668914795, -1.2475998401641846, -0.3189769983291626, 0.7740548849105835, 0.6381967067718506, -0.5189910531044006, 0.4593605101108551, 0.773316502571106, 0.9453734755516052], [-0.8775589466094971, -0.9187382459640503, -0.5094241499900818, -0.33576950430870056, 0.3639453947544098, -1.616464614868164, 2.778794288635254, 1.3525397777557373, 0.4750608205795288, 0.048741765320301056, 1.6309183835983276, 0.0569160170853138, 0.4232271909713745, -1.101260781288147, 0.20932158827781677, -1.3282233476638794, -0.3914427161216736, 0.742676317691803, -0.9810468554496765, -0.19538816809654236, -1.615723729133606, -0.8057589530944824, -0.5024837851524353, 0.3315887451171875, 1.1073817014694214, 1.2199519872665405, -0.44235193729400635, -0.9519596695899963, -0.37480100989341736, -0.6210317611694336, -0.3663500249385834, -1.279200553894043, -0.2128491997718811, 0.09171521663665771, 0.0256052166223526, -0.6312025189399719, -0.37164098024368286, 0.3151082992553711, 0.2093677967786789, 0.2778266370296478, 0.1395806223154068, -0.2803475260734558, 0.751820981502533, -0.970319390296936, 1.948349952697754, -0.7532632946968079, 0.35396185517311096, 1.616392970085144, 0.7906880974769592, 0.6041668057441711, -0.4568193852901459, 1.370819091796875, -0.3044435977935791, 0.34571707248687744, -0.398257315158844, -0.17073224484920502, -0.73673415184021, -1.3000491857528687, 1.1029006242752075, -0.9088062047958374, -0.015145082026720047, -0.2773882746696472, -0.5041791200637817, -0.6182844638824463], [0.1172352060675621, 0.05156558007001877, -2.3826141357421875, -0.9869762063026428, -0.23864729702472687, 0.7493827939033508, -0.12188761681318283, 1.1214648485183716, -0.13906678557395935, 0.023066259920597076, -0.273506760597229, 0.3027508854866028, -2.6428332328796387, 0.14131906628608704, 0.2536396086215973, -0.37358400225639343, -0.6528441309928894, 0.7266069054603577, -1.1340376138687134, 0.3053279221057892, 0.5146104693412781, -1.7715649604797363, -1.6964929103851318, -0.39637765288352966, 0.7607346773147583, -0.3900768458843231, -0.7105863094329834, 0.3825356364250183, 0.18612121045589447, -0.6396470069885254, 0.17906148731708527, 1.1366349458694458, -0.2811450660228729, 1.3919709920883179, 0.4761494994163513, 1.6217516660690308, 0.5991334915161133, 0.6980730295181274, -0.5535948872566223, 2.360935926437378, -1.5122928619384766, 0.722973108291626, 0.061902716755867004, -1.3476834297180176, -0.809354305267334, -0.5664619207382202, 0.6531075835227966, -0.15695396065711975, -0.2110522985458374, -3.272679090499878, -1.2971993684768677, -0.7718679904937744, -0.42996111512184143, -1.6068991422653198, 1.8468666076660156, -0.3271321654319763, 0.8998811841011047, -1.6724525690078735, -0.9032995700836182, 0.5209499597549438, -0.328082412481308, -0.5634551644325256, 1.4227224588394165, 0.013691144064068794], [-0.8087970614433289, -1.1125609874725342, 0.3819234073162079, 0.5581685304641724, 1.349461317062378, 0.25133469700813293, 0.15148267149925232, 0.475730299949646, 2.309997320175171, 0.27040523290634155, 1.3635239601135254, -0.4248751401901245, -0.7073823809623718, -0.9627370238304138, 0.03841901570558548, -0.42465049028396606, 0.8620533347129822, 0.7250973582267761, 0.07366704195737839, -0.21472136676311493, -1.1742273569107056, -0.008106537163257599, 0.511496901512146, 1.8993550539016724, 0.7472545504570007, -1.030758023262024, 1.6758275032043457, 0.7487645149230957, -0.018341418355703354, -1.189374566078186, 0.2977895140647888, -0.9254221320152283, 0.03847305849194527, -1.5111382007598877, -0.5790376663208008, 0.16073814034461975, 0.6924355030059814, 0.9755429029464722, 1.1507842540740967, -0.23907433450222015, 0.27874448895454407, 1.1223704814910889, -0.7548146843910217, 0.15293703973293304, 0.12750020623207092, -0.1261242926120758, 2.0517654418945312, -0.9991444945335388, 0.28531068563461304, -0.27705076336860657, -0.5008499622344971, 0.5414227247238159, -1.293707013130188, -0.7479076981544495, 0.20398208498954773, 0.7368305921554565, 0.008434930816292763, -0.21143648028373718, -2.3218743801116943, -1.1152150630950928, -0.8593639731407166, -1.9247325658798218, 0.990803062915802, -0.6473472118377686], [0.07181411981582642, -0.6519342064857483, 0.3319482207298279, 2.087502956390381, -1.4470939636230469, 0.04311120882630348, 0.5056166648864746, -1.1301127672195435, -1.351857304573059, 0.886946439743042, 1.3307636976242065, 1.3306653499603271, -1.8695605993270874, 2.5331244468688965, -0.600114643573761, -0.12313149124383926, 2.5101735591888428, -0.7815437316894531, 2.5205304622650146, 0.13628363609313965, -1.5777077674865723, 2.0506064891815186, -0.4333832263946533, -1.1588351726531982, 0.08944623172283173, 1.341809868812561, 0.5365514159202576, 0.7201502919197083, -1.1969757080078125, -0.5206983089447021, -0.3670833706855774, 1.5721182823181152, 0.26364707946777344, -0.2856065034866333, -1.5998477935791016, -1.497666597366333, 0.09559670090675354, 1.049782395362854, 0.7606340050697327, 0.5206311345100403, -0.8651037812232971, 0.5896475315093994, 1.1693049669265747, -1.7779414653778076, 2.140326976776123, 0.20744504034519196, -0.8591791987419128, 1.755409836769104, -1.8986343145370483, -2.9495203495025635, -0.8988288640975952, 0.3390250504016876, -0.1168571263551712, 0.5329219698905945, -0.7187793850898743, 0.9328434467315674, 1.1550379991531372, 0.2762061059474945, -0.25969749689102173, -0.2008938491344452, -1.0344390869140625, 2.5171873569488525, -0.26769015192985535, 0.9134128093719482], [-1.4844456911087036, -0.35610395669937134, -0.10965461283922195, -0.18221253156661987, 0.20705267786979675, -0.34546685218811035, -0.04063970223069191, -0.022343313321471214, -0.6170628666877747, 1.4321919679641724, 0.7252956628799438, -0.49257904291152954, 0.6185025572776794, 0.8271253705024719, -0.17988547682762146, -0.8419985771179199, -0.7743129730224609, 1.9901957511901855, -1.0843427181243896, -0.34124523401260376, -0.036197420209646225, -0.7212685942649841, 0.10790996253490448, 0.04163016006350517, 0.49497678875923157, -0.4084993302822113, 0.9521082639694214, 0.27177244424819946, 1.7383310794830322, 1.1864389181137085, 0.09492229670286179, -1.3250354528427124, -0.16960306465625763, -1.095820665359497, 1.5498982667922974, -0.12045329064130783, -0.4742981493473053, -1.1015018224716187, -0.8328718543052673, 1.1351121664047241, -0.47953930497169495, 1.3738147020339966, -0.08282195031642914, -0.08579310029745102, -2.101907730102539, -0.02063450589776039, -1.0872405767440796, 0.7004830241203308, 0.18480068445205688, -1.0386170148849487, 1.1733566522598267, -0.8927441239356995, 0.43378135561943054, -0.2886607050895691, -0.2534105181694031, -1.1143426895141602, 1.4015978574752808, 0.48302140831947327, -1.4354702234268188, -0.9698762893676758, 0.5584094524383545, -0.3340882956981659, 0.11554157733917236, 0.34803882241249084]]], [[[-0.1381947249174118, -1.010888695716858, 1.6456536054611206, 1.043554425239563, 0.641879677772522, -0.47027260065078735, 0.3403353989124298, -1.637318730354309, 0.31059178709983826, -1.0324575901031494, 0.80331951379776, -0.9692904353141785, -1.2482426166534424, -0.9580772519111633, -0.349570631980896, -2.395490884780884, 1.02120840549469, 0.1742871254682541, 2.193549394607544, -0.40297234058380127, -1.4282649755477905, 0.9330501556396484, -0.4895865321159363, -0.1309286504983902, -0.047895535826683044, 0.4803883135318756, 0.00036197411827743053, 0.4494345784187317, -0.8527477979660034, 0.09871921688318253, -0.5783620476722717, 0.4067985713481903, 0.35303765535354614, 1.1843852996826172, -1.3189589977264404, 0.4158176779747009, -0.4868813455104828, 0.35888752341270447, -1.0400264263153076, 0.721124529838562, 0.07362053543329239, -0.009954816661775112, 1.8179506063461304, 0.8250675201416016, -0.11009766906499863, 0.9191710948944092, -0.49459218978881836, 0.2183225005865097, 0.46514394879341125, 2.0810329914093018, 2.092360019683838, 0.42165613174438477, 0.5897051692008972, -1.5766115188598633, -0.3534921705722809, -1.1650294065475464, 1.7173633575439453, 0.046504098922014236, -0.9440734386444092, 0.3013751804828644, 1.4529871940612793, 0.17433778941631317, -0.25926175713539124, 1.1157087087631226], [0.3603856861591339, -0.19759204983711243, -1.4541035890579224, -0.15644890069961548, -0.14616918563842773, 0.8535190224647522, -0.18719595670700073, -0.1969655454158783, 0.3844602108001709, 0.8258597254753113, 2.0108067989349365, -1.2293431758880615, -0.7758119106292725, 1.8345099687576294, -0.07525230944156647, -0.8250781893730164, 0.35499870777130127, 1.5823025703430176, 0.33051419258117676, 0.15427318215370178, 0.3558332026004791, -0.4041898548603058, -1.0460964441299438, 2.1138546466827393, -0.5906659960746765, 0.03172648698091507, 1.42988920211792, -1.124498724937439, -0.7889022827148438, 0.960867166519165, -0.054666101932525635, 0.32112205028533936, 0.2336588054895401, 2.23572039604187, -0.6754054427146912, -0.8338848352432251, 0.7780001163482666, 1.135817050933838, -0.28879469633102417, -0.3203764855861664, -2.0567471981048584, -0.9597899317741394, -0.13978539407253265, 0.006344988010823727, -0.1446458250284195, 0.17440220713615417, 0.3829522728919983, 1.7985862493515015, -0.3145253360271454, 1.0374982357025146, -1.0583378076553345, -0.15215668082237244, -0.4811697006225586, -0.6461989283561707, -0.12981659173965454, 0.7955266833305359, -0.4756702780723572, -1.8531233072280884, 1.0557363033294678, 1.974324107170105, 0.5996620059013367, 0.02336476556956768, -0.1830005645751953, 0.5410553812980652], [2.1879780292510986, 0.6724725365638733, -0.6934752464294434, 0.7672923803329468, -0.8942654132843018, 1.2651159763336182, -0.03594416379928589, 0.7445619702339172, -0.20458011329174042, -2.227382183074951, 0.3187079429626465, -0.5541889071464539, 0.14796197414398193, 1.306410789489746, -1.322993516921997, -0.7481926679611206, -1.166988730430603, -0.05214514210820198, 0.3211866021156311, 0.02934529259800911, 0.433289110660553, 0.884604275226593, 1.2009330987930298, -2.2927305698394775, -1.4063972234725952, 0.01705646701157093, -0.8951613306999207, 0.7812317609786987, 1.2832293510437012, -0.9088362455368042, -0.020795177668333054, -0.19019284844398499, 1.0426193475723267, -0.5563148260116577, -0.15895888209342957, 0.7616363167762756, -0.09478789567947388, -0.7054675221443176, 1.5714842081069946, -0.9216277599334717, -0.7346015572547913, 2.0279526710510254, -0.027256395667791367, 0.9563698172569275, -0.994199812412262, -2.1476123332977295, 0.9094680547714233, -0.11861547827720642, -1.1559230089187622, 0.21687103807926178, -0.17896172404289246, -0.04178827255964279, 0.8629907369613647, -0.7816668748855591, 0.8083857893943787, -0.43306148052215576, 0.7003906965255737, 1.1607213020324707, 1.6878571510314941, 1.198935866355896, -1.2071900367736816, -1.2366849184036255, 1.1824573278427124, -0.6931714415550232], [-0.5483407974243164, -0.6146999001502991, 2.519216537475586, -1.6103076934814453, 0.004777722992002964, 0.0860762670636177, 0.5288235545158386, -0.38092637062072754, -0.2510833144187927, 0.25671687722206116, 0.22421661019325256, -0.2630936801433563, 0.13736124336719513, 0.47631657123565674, -1.1629101037979126, -0.9732142686843872, 1.4627606868743896, 0.18768750131130219, 1.3252322673797607, -0.754182755947113, 0.8083850145339966, -1.015851378440857, 0.8240447044372559, 0.1722269505262375, 1.1671926975250244, -0.9007735848426819, 0.9096366763114929, -0.5721641182899475, -0.8452796339988708, 0.571258544921875, -1.134080410003662, 0.4919615089893341, -0.12199939787387848, 0.506162703037262, 0.14834406971931458, 0.10151249915361404, -0.2596858739852905, -0.5617241263389587, -0.9058637022972107, 0.6702986359596252, -0.255083292722702, 1.753920555114746, 1.0937068462371826, -0.4542454183101654, 0.49546751379966736, 1.2256767749786377, -0.7961490750312805, -1.893797755241394, 0.9757081270217896, -0.7272136807441711, -1.6787720918655396, 0.1761254072189331, -0.3869483172893524, 0.6995066404342651, 1.4556081295013428, 1.7510865926742554, 0.6957346200942993, 0.7486576437950134, -1.0942045450210571, 1.345004916191101, 0.7510249018669128, -0.7552568316459656, -0.2929602861404419, -1.036940097808838], [-0.10376632213592529, -0.42488035559654236, 0.06119374930858612, -1.8599605560302734, 1.1867321729660034, -0.051807042211294174, -0.5871240496635437, -0.13920146226882935, -1.007132649421692, 0.5388684272766113, -0.13234686851501465, -0.751606285572052, 0.16687604784965515, -0.6438098549842834, -0.08488720655441284, -0.5632094144821167, 1.474257469177246, 0.5490497350692749, 0.3353990614414215, 0.5300071239471436, 0.8431238532066345, 1.7612402439117432, 0.07438197731971741, 0.6598042845726013, 0.5806466341018677, 0.8949946165084839, 0.634346067905426, -0.6518761515617371, 0.26683729887008667, -0.6775082945823669, -0.6037220358848572, -0.24994339048862457, -0.8069313168525696, 1.3725370168685913, -0.42962607741355896, 0.9745369553565979, 0.4316500127315521, -0.4374271035194397, 0.9488667249679565, -3.0322108268737793, -0.9825255870819092, 1.8125609159469604, 0.11017833650112152, -0.4304686486721039, 0.7969231009483337, -1.2141494750976562, -1.7004917860031128, -0.6877467632293701, -1.4547467231750488, -0.5658293962478638, -0.3835189640522003, -0.22086840867996216, -0.42439648509025574, -1.6292603015899658, -0.2520989775657654, -1.982629418373108, -0.6938722133636475, 1.7100268602371216, 0.4324895739555359, -1.1039224863052368, 0.19299177825450897, -0.08561372011899948, -0.5560290813446045, -1.3751596212387085], [0.15151169896125793, 0.2434602975845337, 0.785773754119873, 0.4561866819858551, 1.9754674434661865, 0.07401866465806961, -3.5269598960876465, -1.221888780593872, 0.2700630724430084, -1.1179602146148682, 1.2079943418502808, -1.7790645360946655, -0.6838284134864807, -0.6564601063728333, -0.7786406874656677, 1.3596372604370117, -0.6778258085250854, 0.4664056897163391, -0.7531455159187317, 1.309615135192871, 0.8819910287857056, 2.0653650760650635, -1.0113542079925537, -0.07778719067573547, 0.551493763923645, -0.2250053584575653, -0.8699713945388794, -1.7411236763000488, 0.2454504668712616, 0.2149377167224884, -0.06298650056123734, -0.017357345670461655, -0.3924051821231842, -0.4594787061214447, -2.8271641731262207, 0.601815938949585, -0.058316320180892944, 0.22621706128120422, -1.1280720233917236, -0.906105101108551, -0.5881222486495972, -0.07925226539373398, -0.5883989930152893, 0.5542637705802917, 0.29943564534187317, 0.37188076972961426, 0.4066994786262512, -0.03139590844511986, -0.7362465262413025, -0.8253023624420166, -1.644286036491394, -2.0498039722442627, -0.7428134083747864, 1.0202833414077759, -0.4502522945404053, -0.3559405207633972, -0.034260135143995285, -0.4308425486087799, -1.4185696840286255, -1.1634151935577393, 1.5580384731292725, -0.35859158635139465, 1.142473816871643, 0.8004852533340454], [-0.5174044370651245, 1.5506336688995361, 0.5110357999801636, -1.9554469585418701, -0.6696092486381531, 0.4199124574661255, -1.7386000156402588, 0.10796462744474411, 0.8928626179695129, -0.28401118516921997, 0.2130529135465622, -0.16590455174446106, 0.4673893451690674, 0.5103428363800049, -0.3328480124473572, -2.3695619106292725, -0.8476943969726562, -0.4949849247932434, 1.1422886848449707, -0.04283105581998825, -2.1300172805786133, 0.37622493505477905, -0.846796452999115, -0.3750384449958801, 0.6296527981758118, 0.7113891243934631, 2.7826921939849854, 1.1983537673950195, 0.0036916618701070547, 0.8514811992645264, 0.31044965982437134, -1.1405121088027954, -0.23484428226947784, 1.0877807140350342, 0.9097931981086731, 0.5477544665336609, 1.5108529329299927, 0.5317018032073975, 1.0679398775100708, -0.46985703706741333, -0.8187240958213806, 0.3754863739013672, 0.9476084113121033, 0.5764954686164856, 0.620622456073761, 1.62830650806427, 0.1036040261387825, 0.793591320514679, -0.2167225331068039, -0.046653635799884796, 0.8390509486198425, 2.0027880668640137, -0.26363739371299744, -1.0306788682937622, 1.1329033374786377, 1.548959732055664, 0.37427088618278503, -0.24942079186439514, 0.20713143050670624, -0.14288420975208282, -1.4485104084014893, 0.24054470658302307, 1.0481668710708618, -0.526020348072052], [1.4934685230255127, -1.918393611907959, 1.0062718391418457, -1.5745880603790283, -0.47217655181884766, -0.9591631293296814, -0.9813115000724792, 2.2057242393493652, 0.00361064774915576, 1.6808359622955322, 2.0980870723724365, 0.5476152300834656, -0.3094250559806824, -0.3183049261569977, -0.9133190512657166, 0.3735787868499756, 1.2430529594421387, -0.7793294191360474, -0.8237756490707397, -0.04408667981624603, 1.6917836666107178, -0.22032348811626434, 1.6930510997772217, -1.2110638618469238, 0.5003395676612854, 1.6070125102996826, -0.8057965040206909, -0.5293818116188049, 1.0145018100738525, -0.1714615374803543, 0.5417417883872986, 0.6597731709480286, 1.2126590013504028, 1.8490939140319824, -0.8935555219650269, 0.11363638192415237, 1.1567258834838867, 0.8070096373558044, 0.7738924622535706, -0.6375279426574707, 0.6961275339126587, -0.8286320567131042, 1.5739965438842773, -0.26958101987838745, -2.7348530292510986, -2.843966484069824, 0.7632014751434326, 1.8495872020721436, 0.45514237880706787, 0.6343014240264893, 0.2185051292181015, -0.00021515499975066632, -0.31086137890815735, -1.653571367263794, -0.035153016448020935, 0.5835023522377014, -0.42455336451530457, 0.3681681156158447, 0.8050702810287476, -1.6696693897247314, 0.23026694357395172, -0.254885196685791, 1.1802427768707275, 0.6063275337219238], [0.45702821016311646, -1.803636908531189, 0.844356119632721, -0.843498945236206, -0.8400782942771912, -0.36785247921943665, 0.5212110280990601, -0.4955922067165375, 2.060671329498291, 1.801753044128418, -0.5531667470932007, 0.16140824556350708, -0.8455464243888855, -1.369185447692871, -0.992654025554657, 0.8641502261161804, -0.1252424269914627, -0.7145529985427856, -0.6069270372390747, -1.2103816270828247, -0.19077341258525848, 0.47987228631973267, 1.6234309673309326, 0.49736616015434265, -0.3566955029964447, -0.7719172239303589, 2.7637827396392822, -0.3646998405456543, -0.0202327910810709, -1.7627086639404297, -0.30997371673583984, -0.4344753324985504, 0.1900818645954132, 1.2685935497283936, 0.7055661082267761, -0.8991524577140808, -1.5819804668426514, -0.17899231612682343, 0.492775559425354, -0.7202668190002441, -0.42802736163139343, 0.3514484763145447, -1.0021454095840454, 0.6346113681793213, -0.7759697437286377, 1.8846770524978638, -0.02434002235531807, -0.5541463494300842, -0.29208728671073914, -1.088049292564392, 0.24675355851650238, -1.308998703956604, 0.8512901663780212, -0.12723548710346222, -1.3989498615264893, 0.598767101764679, 0.3616858422756195, -1.3116480112075806, 0.44790077209472656, -0.22203952074050903, 0.289366751909256, -1.4789578914642334, 1.1919533014297485, -0.6255374550819397], [0.333749920129776, 1.4728518724441528, -0.10408619791269302, 0.6943622827529907, 1.2830051183700562, -0.9817785024642944, -0.6732786893844604, 0.2888522446155548, 0.9233606457710266, -0.5647050738334656, 1.4293580055236816, 0.9955511689186096, -0.03587799519300461, -0.5648581385612488, 1.8201402425765991, -0.6586771607398987, 0.7385415434837341, -2.3101487159729004, 0.11645790189504623, 0.16511230170726776, -0.20040272176265717, 0.9843433499336243, 0.6201378107070923, -1.2558435201644897, 1.185249924659729, -0.013208632357418537, 1.572298526763916, -0.2571812868118286, -0.1687449812889099, 0.5325621366500854, -0.8824224472045898, 0.4450269043445587, -0.18777428567409515, -1.0126686096191406, 0.9155533909797668, 0.6662111282348633, -1.1765234470367432, 0.8700821399688721, 1.299789309501648, 0.6978880167007446, -0.8059459328651428, -0.33108144998550415, -0.45614880323410034, 0.7944746613502502, -1.5574874877929688, 0.3269787132740021, 0.4845273196697235, -2.010056734085083, 1.239193081855774, 0.4800659418106079, -1.7669016122817993, -0.4182935655117035, 0.8485519289970398, -0.9344991445541382, -1.9082621335983276, -0.9052777886390686, -0.5795618295669556, 0.27629029750823975, 0.6038072109222412, -0.8654646277427673, -2.082045316696167, -0.579657793045044, -0.7515884041786194, -0.05290907993912697], [1.2314743995666504, -1.1729902029037476, 1.6400971412658691, 0.3849848806858063, 1.0456244945526123, -2.3417346477508545, -0.15211234986782074, 0.6733006238937378, -0.531130313873291, -0.12791390717029572, -0.4466612637042999, -0.3967263698577881, -0.52825528383255, 0.9102756381034851, -0.6954976320266724, -0.5215584635734558, 1.2963110208511353, -0.05588262528181076, -0.4482376277446747, 2.5020642280578613, -2.3961124420166016, 0.6187834143638611, 0.1229967400431633, 2.455721616744995, 0.3803538978099823, -0.8127440214157104, -0.18434345722198486, -0.9744108319282532, 1.763518214225769, 0.15738245844841003, 0.228117898106575, -0.23588529229164124, 0.646166980266571, -0.9622747898101807, 0.11936915665864944, -0.4201800525188446, 0.38179436326026917, -0.15483535826206207, 0.32987263798713684, 1.292147159576416, 0.06334470957517624, 0.6779922842979431, -0.38461264967918396, -0.649545431137085, -0.18720197677612305, -0.38024649024009705, 0.16952306032180786, -0.2974901497364044, -0.3701842427253723, -1.0593035221099854, 0.7046212553977966, -0.6509758234024048, -0.41367393732070923, 1.1095768213272095, 1.998611569404602, 1.4478235244750977, 1.0159547328948975, 1.5395406484603882, -0.5828961730003357, 1.296472430229187, 0.7756607532501221, 0.07048378139734268, 0.22201792895793915, 0.5348454713821411], [-1.1599208116531372, 0.8871437311172485, 0.45420581102371216, 0.10602313280105591, -1.2276360988616943, 0.651050865650177, -0.7307875156402588, 1.3873615264892578, -0.30329668521881104, 0.18324504792690277, -2.2757821083068848, 0.677388072013855, -0.10720115900039673, 1.0162253379821777, 1.3063050508499146, -0.9446171522140503, -1.3065649271011353, -0.640247106552124, 0.774061381816864, 0.046341340988874435, 0.4913913309574127, -0.45208755135536194, -0.46349871158599854, -0.8157921433448792, 1.3432273864746094, 1.1058207750320435, 0.6596536636352539, 2.212134599685669, 1.654234766960144, -0.6803878545761108, 1.1553601026535034, 0.7008917331695557, 0.9626690149307251, -0.8523895144462585, 0.6998773813247681, 0.10575052350759506, 1.5863280296325684, 0.3113170564174652, 0.4531663656234741, 1.2708722352981567, -0.43231329321861267, 1.4781829118728638, 1.9055699110031128, 0.395920991897583, -0.11045685410499573, -0.1144433245062828, -0.22452689707279205, 0.5398847460746765, -0.33241358399391174, 2.003830671310425, 0.46424537897109985, 0.8324267864227295, -0.014392365701496601, 0.48725706338882446, 0.6234115362167358, -0.11800512671470642, -0.28849557042121887, -1.438502311706543, -0.994087278842926, 0.12897397577762604, 1.9276483058929443, 0.12535423040390015, -0.41686397790908813, 0.7098380923271179], [0.6318171620368958, 0.23555588722229004, 0.25616979598999023, -1.7417831420898438, 1.5843356847763062, -1.265227198600769, -1.2636888027191162, -1.0653390884399414, -1.0149686336517334, -0.6428592801094055, -0.32534343004226685, 1.137855887413025, -1.1525840759277344, -0.13503585755825043, -0.22047515213489532, -0.20154190063476562, 0.09199944138526917, 1.9562021493911743, -0.2756762206554413, 0.22250770032405853, 0.25261273980140686, 1.4145053625106812, 1.28972327709198, 0.2497473657131195, 0.5443851947784424, 0.41735029220581055, -0.35749781131744385, -0.0014471383765339851, 1.0118188858032227, -1.9551002979278564, -0.44758856296539307, 1.158389925956726, 0.1346183866262436, 0.1719084084033966, 0.2388676404953003, 1.4240175485610962, 0.9205069541931152, -0.8550165891647339, 1.190624475479126, -0.32849037647247314, -0.5276917219161987, -0.24827787280082703, -1.29964280128479, 0.15287061035633087, -0.26303979754447937, -1.6222939491271973, 0.42036768794059753, 1.450477123260498, -1.0999783277511597, -1.1207869052886963, -2.178100824356079, 1.2485156059265137, -0.011144801042973995, -0.8152625560760498, 1.0820852518081665, 0.27824175357818604, 2.645566940307617, -0.1866065412759781, 0.7544766068458557, -0.1388787180185318, 1.1766908168792725, -0.8280033469200134, 1.372958779335022, -0.20308496057987213], [0.1693515032529831, 0.23036104440689087, 0.0753839984536171, -0.3379809260368347, -0.5395922660827637, -0.27986228466033936, 0.4392601251602173, -0.03681855648756027, 0.8722963929176331, 0.041257426142692566, 1.1011477708816528, -0.4502023160457611, 1.1437747478485107, -2.206181764602661, -1.405090570449829, 0.1012272760272026, 0.8671480417251587, 1.4724653959274292, -0.13858802616596222, 1.2291747331619263, -0.02470695972442627, -1.1390926837921143, 1.1387284994125366, 0.4866555631160736, -0.18451784551143646, 0.4240858852863312, 0.38679206371307373, -0.0065453690476715565, -0.027642622590065002, -0.6204506158828735, 0.583794116973877, -1.0846387147903442, -0.6703898906707764, 1.4941751956939697, -0.21019414067268372, 1.0962533950805664, -1.2651764154434204, -1.8944867849349976, 0.5561314821243286, -0.15348899364471436, -0.20372578501701355, -1.200441598892212, 0.5052489042282104, -1.8935153484344482, 0.09751754999160767, 1.4589464664459229, -0.32943102717399597, 0.6081149578094482, -2.1343834400177, -0.9765879511833191, -0.3552951216697693, 0.6440419554710388, -0.6363621950149536, 0.23533189296722412, 0.10079850256443024, -0.6904414296150208, 0.3831084370613098, -1.2361682653427124, 0.477484792470932, 1.2753100395202637, -0.4408130347728729, -0.6975442171096802, -0.41048210859298706, 0.07097875326871872], [0.12036346644163132, 0.027857020497322083, -0.926244854927063, 0.25810620188713074, 1.1367337703704834, -0.3001821041107178, 1.5943522453308105, 0.9836565852165222, 1.00142502784729, 0.4911838471889496, -0.14149878919124603, 1.2003779411315918, -0.15667827427387238, -0.45112746953964233, 1.3256815671920776, 0.6878237128257751, -1.8332717418670654, 1.061937928199768, 0.30619025230407715, -0.23204803466796875, 0.06973563134670258, 0.11592838913202286, -0.8893889784812927, 0.978636622428894, 1.803151249885559, 1.3864381313323975, 0.5151457190513611, -1.1640093326568604, -0.5563866496086121, -1.677516222000122, -0.21263344585895538, 0.035552285611629486, 0.24527597427368164, -1.3859037160873413, -0.320792555809021, -1.5833600759506226, -1.3927323818206787, -0.12313906103372574, -0.6137268543243408, 1.303622841835022, 1.3447312116622925, 0.6052956581115723, 1.1216405630111694, 0.1237095296382904, -2.192207098007202, -0.8572380542755127, 0.11746706068515778, 1.9416508674621582, -0.6600398421287537, -1.1740590333938599, -2.6152873039245605, -0.30711305141448975, -0.18655288219451904, -0.6122815608978271, 0.2057551145553589, 1.3651012182235718, -0.7227754592895508, -1.4828625917434692, 1.191133975982666, -1.1081688404083252, 0.1881529539823532, 1.6552884578704834, 0.047489456832408905, 1.2720025777816772], [-0.2851639986038208, 0.7043618559837341, 0.6555595397949219, 0.9178925156593323, -1.0130956172943115, 0.7635275721549988, -0.910542368888855, -0.36555561423301697, -0.8403845429420471, -1.1576813459396362, -1.1111235618591309, -0.327563613653183, 1.694427728652954, -0.9723818302154541, 1.3977676630020142, -0.8245160579681396, 0.9484590888023376, -1.8738168478012085, 0.5769997835159302, 0.16996149718761444, 1.3055540323257446, -2.1455466747283936, -0.21046370267868042, -1.7801061868667603, 0.6510570049285889, -1.907694697380066, -1.4416487216949463, 0.3865306079387665, 0.47305363416671753, 0.10977110266685486, 0.3809099495410919, -0.6778548359870911, 1.4243677854537964, -0.027751101180911064, -1.8437016010284424, -0.6413598656654358, -0.6379948258399963, 1.1039814949035645, 0.834296464920044, 0.5493330955505371, -1.2533884048461914, 0.7202978730201721, -0.8232497572898865, -0.2826038599014282, 0.3206382393836975, -1.6114169359207153, 0.39319244027137756, -0.3789574205875397, -1.3189369440078735, 1.1936808824539185, 0.24072760343551636, 0.3160307705402374, -1.5271096229553223, 0.7587989568710327, -0.22651559114456177, -1.4444955587387085, 0.6129714250564575, 0.24387216567993164, 0.2719672918319702, 0.6581708192825317, -0.8046095371246338, -0.05397602170705795, 0.25221237540245056, 0.7283226847648621]], [[-1.3032097816467285, -1.4085551500320435, 1.257759690284729, -0.3903457522392273, -2.0043420791625977, 0.6879453659057617, -0.8676369786262512, -0.4051544666290283, 0.08926921337842941, -0.8459975123405457, -1.8993531465530396, -0.9174767136573792, -0.10938658565282822, -1.3445030450820923, -1.2883301973342896, 1.5708588361740112, 0.28874149918556213, -0.22959227859973907, -0.7086868286132812, 0.39481720328330994, 0.26903894543647766, -1.6405611038208008, -1.0465071201324463, -0.02242831327021122, 0.5132469534873962, 1.0133219957351685, 1.1499049663543701, -0.257793128490448, -0.343418687582016, 0.46145230531692505, -1.4985921382904053, -1.092057466506958, 0.9258062839508057, -1.5086678266525269, 0.26615577936172485, 0.2800973057746887, -1.6301958560943604, 0.04852263256907463, 0.5525507926940918, 1.4505118131637573, 1.5981765985488892, 0.0864635780453682, -0.22426117956638336, -0.006411214359104633, -0.9308244585990906, -0.8712772727012634, -0.9833381772041321, 2.0533626079559326, -0.0013330749934539199, 0.019905518740415573, 1.6542667150497437, -0.8342505097389221, 0.6940916776657104, 0.9622942805290222, 0.16763071715831757, -0.008901608176529408, 0.9018924236297607, 1.7586411237716675, -0.34705209732055664, 2.8943638801574707, 1.6327780485153198, -1.6984455585479736, -0.025865139439702034, -0.5131883025169373], [1.322936773300171, 2.2946534156799316, 0.7409218549728394, -0.1232469230890274, -0.6332864761352539, 1.141118049621582, -0.8617064952850342, 2.272268056869507, 1.1405768394470215, 2.0610954761505127, 0.2042611986398697, 1.9521714448928833, -0.26406678557395935, 0.39606937766075134, 0.7642431855201721, -0.32181671261787415, -0.2428097277879715, 1.7416540384292603, -1.0879478454589844, 2.13153338432312, -0.3908078968524933, -0.6114645004272461, -1.4880280494689941, -1.4631530046463013, -1.0565649271011353, 0.043497323989868164, 0.47842511534690857, 0.567094087600708, 0.10118638724088669, 0.017955457791686058, 1.528372049331665, 0.13258656859397888, 0.15945599973201752, -0.11496848613023758, -0.6673130393028259, -0.286099374294281, -0.39111945033073425, -0.13604110479354858, 3.5178210735321045, 1.3668889999389648, -1.01339590549469, -1.4137153625488281, -0.5022480487823486, 0.07745496183633804, 0.38862141966819763, 0.1477193534374237, -1.6568145751953125, -1.7813773155212402, -0.5138444304466248, -0.7832794785499573, -2.285306930541992, -1.442802906036377, 0.1341405212879181, -1.2943392992019653, -0.5389161109924316, 0.33881667256355286, 1.2698975801467896, -0.9180983304977417, 0.0445450097322464, -0.09201318770647049, -0.758849024772644, -0.7403687238693237, 1.3833351135253906, -0.6452460289001465], [-0.03079674020409584, 0.13655142486095428, 1.6838027238845825, 0.08240833133459091, 1.7578758001327515, 1.1219369173049927, -0.6138486266136169, 0.7913243174552917, -0.8650104403495789, -0.11599523574113846, -0.39066535234451294, 1.4669184684753418, -0.8504008650779724, 0.9801990389823914, 0.22092624008655548, 0.8349664211273193, -2.3756494522094727, -0.19557863473892212, -0.17326252162456512, 2.5137205123901367, 1.9150186777114868, -0.08509179949760437, 0.07335860282182693, -1.9273062944412231, 0.2520577013492584, -0.9186553359031677, 1.2402271032333374, -1.2311217784881592, 0.20422431826591492, -0.062390413135290146, -0.3569563925266266, -0.4003745913505554, -0.3122195601463318, -1.5240541696548462, 0.5995477437973022, 1.2585855722427368, -1.0156769752502441, -0.7585468888282776, -1.253064513206482, 0.24470211565494537, 0.11376633495092392, -1.3101149797439575, -1.72486412525177, -0.14190854132175446, 1.0808608531951904, 1.3752336502075195, -0.3895503282546997, -0.38962069153785706, 1.8238861560821533, 1.419551134109497, -0.8965878486633301, -0.989684522151947, -1.7230662107467651, 0.49143263697624207, -0.330195814371109, -0.9048953056335449, -0.3653501868247986, -1.3306965827941895, 1.1798471212387085, -0.08967771381139755, 0.7065779566764832, -1.6369787454605103, -0.8827138543128967, 0.6572413444519043], [0.5688979625701904, -0.11750917881727219, -1.0150200128555298, -0.7312965393066406, 0.5528442859649658, 0.004776351619511843, -0.337612122297287, 0.6417142152786255, -1.5046929121017456, -0.7990649938583374, 0.02153613604605198, 1.5040756464004517, -0.6577517986297607, 1.2229238748550415, -0.44088634848594666, 0.854735255241394, 0.7745774388313293, 1.3506625890731812, 0.6358617544174194, 0.5600656867027283, -0.1689896285533905, 1.217393398284912, 1.1514263153076172, -0.3629404902458191, -1.741645097732544, 1.0914721488952637, 0.39496731758117676, -0.4955832362174988, -0.7303245663642883, 0.16983316838741302, -0.5634880065917969, 0.5954596400260925, -0.25581124424934387, 0.3867185115814209, -2.7940945625305176, -0.7531731128692627, -1.579795241355896, -0.8507612347602844, 0.39931389689445496, -0.06512415409088135, 0.4533911943435669, -0.6453508734703064, -1.4382553100585938, 0.3185591697692871, -0.719753623008728, 0.5961838364601135, 0.7761504650115967, 0.5233759880065918, -2.044419527053833, 0.31264108419418335, -1.1439011096954346, 0.897842526435852, -0.37191152572631836, 0.23723691701889038, 0.8707143664360046, -0.008719327859580517, 1.4424506425857544, -1.2584372758865356, -0.8724331855773926, 2.4495232105255127, 0.818223237991333, 1.7007948160171509, 0.46954309940338135, -0.524945855140686], [1.4943585395812988, 1.3859496116638184, 0.3656843304634094, -0.5679981708526611, 0.6360520124435425, 1.3802399635314941, -1.0064877271652222, -0.37931764125823975, -0.49375221133232117, 0.6938718557357788, 1.6112068891525269, -0.6629020571708679, 0.5224304795265198, 1.7981646060943604, -0.6019996404647827, 1.2324695587158203, -0.8681746125221252, -1.059667706489563, -0.2025073915719986, 1.0290621519088745, 0.04572214186191559, -0.8583725690841675, -1.4162273406982422, 0.6977763772010803, 1.9925379753112793, 1.6692208051681519, 0.04203606769442558, -0.7261157035827637, 0.44619086384773254, -1.0943267345428467, -0.8867244124412537, -0.7558334469795227, 0.5592957139015198, 1.9739632606506348, -0.7574946284294128, -0.9221000671386719, 1.2186360359191895, -0.8044843673706055, 1.353046178817749, -0.531865656375885, -1.1196763515472412, 0.115109384059906, -0.7663249373435974, -1.4434605836868286, -1.1690500974655151, -0.4480040967464447, 0.6453943848609924, 1.3806445598602295, -0.6948428750038147, -0.14362700283527374, 0.410291463136673, 1.4218358993530273, 0.051429346203804016, -1.1981643438339233, 1.1185364723205566, -0.9789149761199951, 0.19189682602882385, 0.5995882749557495, 1.2332159280776978, -0.42542779445648193, -0.6731253862380981, -0.5597110986709595, -0.85722416639328, 2.3743410110473633], [0.09817852824926376, 0.33089372515678406, -0.9017242789268494, 0.279610812664032, 0.5077676773071289, 0.10001464933156967, 1.4862741231918335, -0.17130224406719208, 1.3127951622009277, -0.31140777468681335, -0.8582549095153809, 0.33686861395835876, -1.8804869651794434, -1.327035903930664, -0.09305224567651749, -0.860805332660675, -0.22864720225334167, 0.733532190322876, 0.9935156106948853, 0.7794314026832581, 0.7016611099243164, -0.901523768901825, -0.35460761189460754, -0.8750571012496948, -1.1727598905563354, -1.1453853845596313, 1.8551961183547974, -0.8303028345108032, 0.16991718113422394, 0.7002075910568237, 0.38279834389686584, -0.7044036984443665, 0.17961134016513824, -1.5431348085403442, -1.2323243618011475, -0.21130584180355072, 0.7673062086105347, 1.0276182889938354, 0.7370584011077881, 0.24671146273612976, 0.47319960594177246, 1.4297953844070435, -0.9097868204116821, -0.4838130474090576, -1.0822988748550415, -0.8057454824447632, -0.66303551197052, 0.36074841022491455, -0.903481125831604, 0.5527318716049194, -0.7162327766418457, 1.208599328994751, 1.0143612623214722, -0.6364022493362427, -0.5184544324874878, -0.1063094213604927, 0.4910375475883484, -0.20879819989204407, 0.7772449254989624, -1.1250687837600708, -1.0731416940689087, 0.849738597869873, 0.6985753774642944, 0.20334933698177338], [-0.6620802283287048, -0.5730706453323364, 0.03538592904806137, -0.8739436864852905, -0.15587061643600464, -0.048891469836235046, 2.1790990829467773, -0.3007460832595825, -0.6727169156074524, -0.5264837741851807, 1.2350234985351562, -1.5482386350631714, 0.7112827897071838, 0.45029687881469727, 0.4980895519256592, 1.2091999053955078, -0.8207481503486633, 0.7706156969070435, -0.9289768934249878, 0.8383402228355408, 0.1318904459476471, 1.18648362159729, -0.38775256276130676, -1.2470431327819824, 0.16811025142669678, -0.20575930178165436, -0.3801710605621338, 0.7921314239501953, 0.5090847015380859, -0.42780253291130066, 0.5077677369117737, 0.666936457157135, 1.4955263137817383, 0.560651957988739, -0.2883415222167969, -1.4537112712860107, 1.1537424325942993, -1.1560792922973633, -0.9624719023704529, -0.08769858628511429, -0.2942228615283966, -1.4847935438156128, -2.2408082485198975, 0.472007155418396, 0.04020201787352562, 0.23302724957466125, 2.6821818351745605, -0.2150619626045227, 0.4156235456466675, 1.0855635404586792, 1.2391307353973389, 0.3678038716316223, -1.3621903657913208, 0.2967628538608551, 0.2205335795879364, 0.431503564119339, -1.6893913745880127, 1.7459454536437988, -0.3218272030353546, -0.2725972831249237, 2.2841601371765137, 2.3284242153167725, -0.1138525977730751, 0.6938652396202087], [-1.5313377380371094, -0.14434903860092163, -0.3100528419017792, -2.0346570014953613, 1.3752696514129639, -1.9336069822311401, 2.669997215270996, -1.4494987726211548, -2.5147669315338135, 0.0635421872138977, 1.1948732137680054, -1.9763400554656982, 0.10582486540079117, 0.050295159220695496, -0.8476631045341492, 0.03863675147294998, -0.13360798358917236, -0.17139476537704468, -0.38250118494033813, -0.7066915035247803, 0.34523507952690125, 0.6779600977897644, -1.1312785148620605, -1.2321171760559082, 1.9534449577331543, -0.4115036725997925, -0.8824042677879333, -1.768646001815796, 1.1059203147888184, 0.42086073756217957, -0.8888649344444275, 1.4526221752166748, 1.1055846214294434, -1.6256426572799683, -0.6045635938644409, 0.29445263743400574, 0.50991290807724, 0.921759843826294, -2.2590601444244385, -1.1493407487869263, 0.39625540375709534, 0.4590328633785248, 0.6908928155899048, -0.5617008209228516, 1.0803275108337402, -0.12230786681175232, 0.42945167422294617, 0.9230348467826843, -0.14984656870365143, 0.6615203022956848, -1.638252854347229, 0.7400050163269043, -0.12130674719810486, 0.11711329221725464, -0.5882457494735718, -3.097338914871216, 0.813178539276123, 0.6506825685501099, 1.6390595436096191, -1.7466803789138794, -1.781825065612793, 0.14213083684444427, -0.6058157682418823, -0.3609592616558075], [1.4323318004608154, 0.7512513399124146, 0.5924474596977234, 1.1745926141738892, -1.3091648817062378, -0.3628794550895691, 0.6658983826637268, -0.7406931519508362, 1.1506378650665283, -0.7294014692306519, 1.3443851470947266, -2.0875155925750732, 0.8008920550346375, -0.24247108399868011, -0.44119325280189514, 1.0293747186660767, 1.0320963859558105, -0.5896912813186646, -1.072398066520691, 1.1201918125152588, 0.3943588137626648, 2.145477771759033, 0.683037281036377, 0.5574367046356201, 0.12032121419906616, -1.047497272491455, -0.17631778120994568, 1.8107913732528687, 0.17675244808197021, -1.0245765447616577, -0.03140632435679436, 0.6935263276100159, -0.41132694482803345, -0.7475412487983704, -0.7960383892059326, 0.49656105041503906, 0.3263842761516571, 1.3564541339874268, 1.1271135807037354, 0.9369884729385376, -1.697670817375183, -0.005952952895313501, 0.46136611700057983, 0.22253109514713287, 0.6086367964744568, -0.5710323452949524, -0.26840120553970337, 0.3908405005931854, 1.226071834564209, 1.1291624307632446, 0.9716792702674866, -0.4441337287425995, 1.8576126098632812, 1.1773123741149902, 0.5504891276359558, 0.9419156312942505, -0.3630301356315613, -1.0084975957870483, -2.1579198837280273, 0.9621029496192932, 0.5304996371269226, -0.20262204110622406, 0.2995777726173401, -1.350258231163025], [1.5229328870773315, 1.6111146211624146, -0.6478345394134521, -0.5952853560447693, -1.180929183959961, 0.11376428604125977, -1.9019720554351807, -0.8296894431114197, -0.41305381059646606, 0.7210129499435425, -0.33188024163246155, 1.6360758543014526, 1.4564529657363892, 0.6549007296562195, -0.18007223308086395, 0.8085372447967529, 0.8764578700065613, 2.0512213706970215, -0.9600472450256348, 0.7018045783042908, 0.033134881407022476, 0.6989873647689819, -0.24011921882629395, -0.46202778816223145, 1.747852087020874, -1.0188277959823608, 0.5159691572189331, -0.5803763270378113, 1.3906632661819458, -0.1391303390264511, 0.24579183757305145, -0.7408368587493896, -0.7420985102653503, -2.100783348083496, 1.4571717977523804, 0.8626437187194824, -1.273447871208191, 0.5031960606575012, -0.8659777641296387, -0.1915934532880783, 1.0965425968170166, 1.2512080669403076, 1.443078875541687, 0.8904556632041931, -0.5764138102531433, -0.8496009707450867, 0.890821099281311, -0.1666364073753357, 0.0016675482038408518, -2.0962870121002197, -2.0646004676818848, -0.6799240112304688, 1.8145722150802612, -1.369937777519226, 0.6237272620201111, 1.9421323537826538, -0.41675177216529846, -0.6658526659011841, -0.14205294847488403, 1.4440124034881592, -0.12104585766792297, -0.4839646518230438, -0.3991847634315491, 0.30141133069992065], [2.3684751987457275, -0.5446182489395142, -0.6700097918510437, 1.4215961694717407, -0.559655487537384, -0.5201317071914673, 0.0106126070022583, -0.10589422285556793, -0.795382022857666, -0.4811779260635376, -0.8093088865280151, 1.533994436264038, -0.47339069843292236, 0.6294263005256653, -0.7920819520950317, -0.6913578510284424, -0.6392260193824768, 0.6939657926559448, 0.01445859670639038, 1.1621384620666504, 2.1208949089050293, 1.0330026149749756, -0.24970300495624542, -0.024208681657910347, -0.20064771175384521, -1.583141565322876, 0.19112852215766907, -2.236741304397583, 0.7816398739814758, 0.6169225573539734, -0.02442581206560135, 0.5657406449317932, -1.1765552759170532, -0.7629585266113281, 0.03470088168978691, -0.21661284565925598, -1.6693459749221802, 1.0703611373901367, -0.5966671109199524, -0.5375010371208191, -0.5978214740753174, -0.23653090000152588, -1.2951624393463135, 0.4981583058834076, 0.6651197075843811, 0.3790356516838074, 0.025634106248617172, 0.32540127635002136, 0.4395900368690491, 0.535282552242279, 0.9224082827568054, 1.040570616722107, 1.7611503601074219, 0.34864741563796997, -2.202754497528076, -1.6882834434509277, 0.5341277122497559, -0.8970386385917664, 0.6910337805747986, 0.7941861748695374, 0.7986721992492676, -1.4317141771316528, -0.2731330990791321, 0.6983967423439026], [-0.34710457921028137, 0.020176760852336884, -0.5874713659286499, 0.45921415090560913, 0.8414655327796936, -0.670353889465332, 2.159644842147827, -0.04972940683364868, 0.1521439552307129, -0.10102448612451553, 1.3312054872512817, 0.5918578505516052, -0.20944498479366302, -1.2884554862976074, 0.04213104769587517, -1.9214138984680176, 0.2680225670337677, -2.047983407974243, 1.005868911743164, 0.19277630746364594, -0.3098270893096924, 2.0919058322906494, 1.1699663400650024, -0.7442309856414795, -0.23393619060516357, 1.194966435432434, 0.38554301857948303, -0.1285402774810791, -1.6648998260498047, 0.7819913029670715, 1.5523760318756104, -1.4838582277297974, 0.25858786702156067, 0.2722376585006714, 2.6130130290985107, -0.15867258608341217, 1.4972349405288696, 1.3831311464309692, 0.6628785133361816, -0.5480559468269348, 0.7039471864700317, 1.0801948308944702, 1.3100123405456543, -0.9969757199287415, -1.3041446208953857, 1.4988011121749878, 0.3860509693622589, -1.36678946018219, -0.35657742619514465, -0.36513638496398926, -0.4942774772644043, -1.428418517112732, -0.796106219291687, -0.08413013815879822, 0.3525034785270691, 1.1941921710968018, -1.6576683521270752, -0.4337424337863922, -1.270621657371521, 1.0430867671966553, 0.04871524125337601, -0.24822275340557098, -0.2634138762950897, 0.9749042391777039], [0.4066787660121918, -0.41022422909736633, 0.23977039754390717, 2.1923375129699707, -1.1673918962478638, -0.00797722302377224, -0.7729182839393616, -0.32192444801330566, -0.578561544418335, 0.8065722584724426, -0.052181657403707504, 2.1425938606262207, -0.17818811535835266, -1.1826813220977783, -0.041667744517326355, 1.1177383661270142, -0.6478496789932251, 1.1941847801208496, -0.9677315354347229, -0.11438731849193573, 1.124487042427063, -0.6151832342147827, 0.3933275640010834, 0.8664956092834473, 0.5048004388809204, 1.9936885833740234, -0.6129202842712402, 1.2832386493682861, -0.17715021967887878, 0.48916515707969666, 0.41198256611824036, 0.3237835168838501, 1.117295265197754, -0.08623084425926208, -0.7218317985534668, 2.3356049060821533, 0.33601945638656616, 0.9284265041351318, -1.3149200677871704, 0.5316857695579529, 0.7784398198127747, -0.15123245120048523, -0.1667882204055786, -0.3679163157939911, -0.6770721077919006, 1.3377360105514526, -0.8488796353340149, 1.306244969367981, -0.6521270275115967, 1.7272192239761353, -1.0895313024520874, -0.6570799350738525, -0.2755638062953949, 0.3187102675437927, 0.4833146035671234, 0.5557458996772766, -2.4978139400482178, 0.47631222009658813, 2.489300489425659, -1.3662949800491333, 0.17564430832862854, -0.623250424861908, -0.6999706029891968, 0.4940539300441742], [0.6474714875221252, -1.458259105682373, -0.8381859064102173, -0.3710425794124603, -1.6690171957015991, 0.014500979334115982, 1.7682559490203857, -0.36216187477111816, 0.46851077675819397, -0.39909473061561584, -0.0848674327135086, 0.15711665153503418, 2.750246047973633, -0.6468931436538696, -1.746946930885315, 0.388652503490448, 0.8857260346412659, -1.0004948377609253, -1.8098112344741821, -0.9241867661476135, -0.7197577953338623, -0.9863176941871643, 1.0531669855117798, -1.8100579977035522, -0.24361418187618256, -0.6247897148132324, -0.24769896268844604, -0.7187947630882263, -1.9559406042099, 0.058708593249320984, -0.42018574476242065, 0.2411983460187912, -0.6186816096305847, 0.765960693359375, -1.0780318975448608, 2.5433642864227295, -1.145555019378662, -1.5251094102859497, 1.0468111038208008, 0.6078778505325317, -0.27432966232299805, 0.2660180330276489, -0.7052434086799622, 1.741920828819275, 0.16572165489196777, -0.4247802793979645, 1.0099595785140991, -0.10072405636310577, 0.6857130527496338, -0.21530455350875854, 0.7125222682952881, 0.4491751194000244, 0.2925553321838379, -0.7474978566169739, -0.1419709473848343, -0.15498727560043335, -0.20681273937225342, 1.5623929500579834, -0.7248813509941101, 0.6488111019134521, 0.24015355110168457, 1.0874630212783813, 0.286104291677475, 0.8777666091918945], [-0.5463201999664307, 1.3739720582962036, 1.054800271987915, -0.3960651159286499, -0.6952677369117737, -2.2164905071258545, -0.4102153778076172, 0.24591045081615448, -0.19533267617225647, -2.3046040534973145, 2.0516738891601562, 0.26952242851257324, 1.4775773286819458, 0.7110602855682373, 0.37699684500694275, -0.23777350783348083, -0.6736634373664856, -0.15380512177944183, -0.34410956501960754, -0.33927369117736816, 0.9638551473617554, 0.15494056046009064, 0.822485625743866, -0.997615396976471, -0.6516914963722229, -1.0046638250350952, -1.139099359512329, 0.8672753572463989, -0.046773653477430344, -0.916985273361206, 0.19797854125499725, 1.1969257593154907, -0.990719199180603, -1.4810082912445068, -0.22017335891723633, -0.06410744041204453, -1.6877992153167725, -0.47492048144340515, 0.7843524217605591, 0.19649924337863922, 0.4467700719833374, -0.5198322534561157, 0.28188371658325195, 0.17480146884918213, -0.34260252118110657, 0.3774731457233429, 2.374725580215454, -2.168184995651245, -0.7602468132972717, 2.6054446697235107, -1.2110389471054077, -1.059645175933838, 0.11728503555059433, -1.0746711492538452, 0.6168954968452454, -1.983120083808899, -1.1670562028884888, 0.6836426854133606, -0.4910469949245453, 1.1168850660324097, -0.22545191645622253, -0.023151863366365433, -2.1799747943878174, 1.5502985715866089], [1.7139978408813477, 0.28193846344947815, 0.6154540181159973, -0.5871652364730835, 1.4189869165420532, 1.5979818105697632, -0.6020200252532959, -0.2547553777694702, 1.1118916273117065, -1.691437840461731, -1.1281118392944336, 1.2314176559448242, 0.9044811725616455, 0.6444177031517029, 1.0304926633834839, -0.7073022127151489, 1.109297275543213, 0.5379525423049927, 0.6918199062347412, -1.4364697933197021, -0.5809089541435242, -0.9631006717681885, -0.7850906252861023, 0.8846383094787598, 0.6126309037208557, 0.04125034436583519, -1.669087290763855, 0.4026234447956085, -0.12652206420898438, -0.6524174213409424, -0.6087422966957092, -0.9631116390228271, 0.3898312449455261, -0.7631741762161255, -0.2939082682132721, -0.5504720211029053, 1.256591558456421, 0.49522241950035095, 0.2625482976436615, -2.1530725955963135, -0.5957891941070557, -0.8056286573410034, 0.12025748938322067, 2.734497308731079, -0.12045285105705261, 0.5221111178398132, -0.13785387575626373, 0.9483235478401184, 0.40543046593666077, 1.21896493434906, 1.1840108633041382, -0.14435529708862305, 1.159456491470337, -2.579475164413452, 0.6744614243507385, -1.5058737993240356, 0.6543065905570984, 1.2591979503631592, -0.8548455238342285, -0.7542569637298584, -0.6470530033111572, 0.7661840915679932, -1.7947388887405396, -0.32181882858276367]], [[1.1089736223220825, -1.4966905117034912, 0.26267626881599426, -1.1504313945770264, 0.5260435342788696, -1.4305561780929565, -1.2190099954605103, 0.10373867303133011, 0.4356559216976166, 0.4561612606048584, 0.2516825199127197, 0.06111731007695198, -0.05763949081301689, 1.1308751106262207, 0.37980735301971436, 0.9276840686798096, -1.4127823114395142, 1.4554202556610107, -1.8397250175476074, -0.306706964969635, 1.1478244066238403, -0.6390500664710999, -0.8767624497413635, 0.14564037322998047, 1.5071772336959839, 1.3020455837249756, -1.5370947122573853, 0.40519464015960693, -0.4848625957965851, -1.338765263557434, -0.8611506223678589, -1.3161488771438599, -1.0379419326782227, -1.4677624702453613, 0.9860682487487793, -0.18763242661952972, 0.7609942555427551, 0.6059611439704895, -1.4948676824569702, 0.6895233988761902, -1.3509465456008911, -1.0561649799346924, 0.19055067002773285, -0.4862412214279175, -0.5373337268829346, 0.7266417741775513, 0.015275323763489723, -3.3660192489624023, 1.6658525466918945, 2.328258991241455, 0.6641473174095154, 0.9562974572181702, 0.37799787521362305, -0.9192537665367126, 0.07923293858766556, 0.26200422644615173, 0.38079994916915894, 0.45206424593925476, -0.38807693123817444, 0.7657690644264221, 1.9611879587173462, 1.4589953422546387, 0.5660702586174011, -0.6689872741699219], [-0.8076024055480957, 0.035615961998701096, -1.1270172595977783, 0.11520783603191376, 0.10455857962369919, 0.8073688745498657, 1.2907508611679077, 0.5133639574050903, 0.38884520530700684, 0.44267934560775757, -0.6165624260902405, 0.5415668487548828, 1.567541480064392, 0.9019259810447693, -0.09666869044303894, 0.3073421120643616, 0.961487352848053, 0.8831873536109924, 1.1343377828598022, -1.6114563941955566, 0.8919211626052856, 0.6837582588195801, 0.11574114859104156, 0.253122478723526, -0.6976677775382996, -0.415179967880249, -0.19000402092933655, -0.8743863105773926, -0.5307075381278992, 0.3762577176094055, -0.017546091228723526, -1.3900467157363892, 2.3491435050964355, -2.7422468662261963, 0.6098591685295105, -0.649608314037323, 0.28640016913414, 0.5807389616966248, 0.40314429998397827, 0.16653943061828613, -0.11807642877101898, 1.4532263278961182, -0.9525416493415833, 1.3220391273498535, -0.5026089549064636, 0.25946134328842163, 0.6554285287857056, -0.008382970467209816, -0.9693905711174011, -0.30868101119995117, 0.10214236378669739, -1.2323209047317505, 0.12811331450939178, -1.1923456192016602, -0.7562195062637329, 0.35392332077026367, -1.0573796033859253, 0.9889820218086243, 0.21624131500720978, -1.278340458869934, -0.9248111248016357, 0.8924088478088379, 1.7778574228286743, 1.1761131286621094], [-1.29912531375885, 1.0397562980651855, 1.8455660343170166, 1.5177619457244873, 0.45797300338745117, 1.0638383626937866, 0.5897607207298279, -0.3122597932815552, -0.614721953868866, -1.7305086851119995, 0.7111961245536804, -1.1889925003051758, 0.7068661451339722, 1.6285370588302612, 0.3880564272403717, -0.30012238025665283, -0.19282753765583038, 1.2023625373840332, -1.1577907800674438, 1.7606979608535767, -0.6707615852355957, -0.5122107863426208, 1.6570302248001099, -1.5560276508331299, -0.6073276400566101, 0.7571117877960205, 0.676876962184906, 0.41131842136383057, 0.023611335083842278, -0.8010706901550293, -0.023702245205640793, -1.20698082447052, -0.03840845450758934, -0.4900442659854889, -0.016301779076457024, 0.7302382588386536, -1.7679646015167236, 0.2685128152370453, -0.05490203574299812, -0.9616194367408752, 0.9513206481933594, -0.6213886737823486, -0.08354198932647705, -0.6447240114212036, -2.7731168270111084, 0.3686036169528961, -0.32158225774765015, 2.0450122356414795, -0.2747795581817627, -1.392232060432434, 1.3660892248153687, -0.5495470762252808, -0.6794924139976501, -0.4128171503543854, 1.8916252851486206, 1.830984115600586, -0.4616556763648987, 0.05579136684536934, 1.335412859916687, -0.40489867329597473, 0.4336412250995636, 0.04731478542089462, 2.0680291652679443, 1.6066677570343018], [-0.12214670330286026, -0.7919251322746277, 0.24324552714824677, -0.05126709118485451, -1.3562723398208618, 0.005699239205569029, 1.0110392570495605, -0.373879075050354, 0.008183377794921398, -0.7863246202468872, 0.04948268458247185, 0.16887503862380981, 0.2894389033317566, -1.611111044883728, -0.6587757468223572, -1.3861368894577026, -0.3405214548110962, -1.2300840616226196, 0.8675264120101929, -0.780735969543457, -0.0006725532002747059, 2.258315324783325, -0.550434947013855, 0.10967258363962173, 0.34237831830978394, 0.5245143175125122, 0.009562545455992222, 0.6920536756515503, -0.32493311166763306, 0.7356528043746948, 0.2446100264787674, 1.3066390752792358, 0.950234055519104, 0.10460943728685379, 0.09227383136749268, -0.6789006590843201, 0.7356429696083069, 0.20925860106945038, -1.6640199422836304, 1.4323770999908447, -1.4044110774993896, -1.191109538078308, 0.3429482579231262, -1.4778715372085571, 0.26543787121772766, 0.062443945556879044, -0.11493179947137833, -1.6052064895629883, 0.03821765258908272, -2.1469035148620605, 1.6418367624282837, -1.0616849660873413, -0.6762055158615112, 1.5797030925750732, 1.0260612964630127, -0.6350606083869934, -1.5072011947631836, 1.2646499872207642, -0.21572455763816833, -0.6246398091316223, 1.3761872053146362, -0.3702092468738556, 1.175845742225647, -2.0704312324523926], [-0.5546644330024719, 1.5097236633300781, 0.989252507686615, -0.01699831150472164, -0.4879954159259796, 1.1259669065475464, 1.6114673614501953, -1.0359935760498047, -1.6215298175811768, -0.0827474445104599, 0.31028980016708374, -0.16362448036670685, -1.5937552452087402, -0.6610833406448364, 0.4765526056289673, -1.2708698511123657, 1.0405027866363525, -0.12825217843055725, 0.3806227743625641, 0.011024721898138523, 0.21507461369037628, -2.079185724258423, -0.8283197283744812, -0.23902733623981476, -0.0002458461676724255, 0.08926176279783249, 1.0240895748138428, -1.0091017484664917, -1.524401307106018, -1.9127174615859985, -1.0173908472061157, -0.24509039521217346, 0.21540570259094238, -0.07634530961513519, -0.7934882640838623, 0.7244082093238831, -0.26726964116096497, 0.20690155029296875, -0.8830880522727966, 0.7379052639007568, -0.36337921023368835, 1.3179936408996582, 0.3532008230686188, 0.07871490716934204, -0.2355053871870041, -0.1569671779870987, -0.67653888463974, 1.1562159061431885, -0.37334078550338745, -1.3027399778366089, -0.631951093673706, 0.9922521710395813, 0.20326310396194458, 0.07214754819869995, -1.2179276943206787, 0.6859675049781799, -0.19596132636070251, -0.31891930103302, 0.22438156604766846, -0.5824577808380127, -0.8692101836204529, -2.0258045196533203, 1.3368855714797974, 0.6808653473854065], [2.0221283435821533, 2.3989109992980957, 0.5139979720115662, -0.5332359671592712, -0.2144361287355423, -1.1944025754928589, -0.6488991379737854, -0.05598565936088562, -0.7452701926231384, -0.04885204881429672, -0.32652562856674194, -0.0692167580127716, -1.778840184211731, 0.6877334713935852, -0.6659417748451233, -0.16863180696964264, 1.2406210899353027, -1.5894428491592407, 0.7149814963340759, 0.39176666736602783, -0.03418392315506935, -0.39394810795783997, -0.22981402277946472, -0.480991393327713, 0.626045823097229, -0.4373062252998352, -0.36754050850868225, -0.44221580028533936, 1.539339303970337, 1.8843706846237183, 0.8898724913597107, 0.5872921943664551, -0.9337216019630432, 0.520089864730835, 0.4344137907028198, -1.5092869997024536, 0.18399296700954437, -0.4151821434497833, -0.855750560760498, -0.09846324473619461, 1.7125366926193237, 1.0890134572982788, -0.803945004940033, -0.9618098735809326, 0.48896506428718567, -2.1333541870117188, 0.661486804485321, -2.037838935852051, 0.11291516572237015, 0.9608744382858276, -0.2596953511238098, 2.542245388031006, 0.5728853940963745, -0.32126978039741516, -0.8364658951759338, -0.899659276008606, 0.17062555253505707, 1.734087347984314, -0.866679847240448, -0.6859957575798035, 0.7364252805709839, -0.25666677951812744, -0.9248437881469727, -1.1556333303451538], [-0.7021802663803101, -0.3420594036579132, 0.9046680927276611, 1.218825340270996, -0.32920774817466736, -1.4810932874679565, -0.005283698905259371, 0.6381887197494507, -0.3106062114238739, -0.896312415599823, -1.53032386302948, 0.6576551795005798, 0.15625527501106262, 0.7129688262939453, 0.9949472546577454, 1.5265520811080933, 1.9553539752960205, 0.016024645417928696, -0.9081898331642151, 0.8374579548835754, 0.41405048966407776, 0.773463785648346, -0.6417446732521057, 0.05329355224967003, 0.008012956939637661, -0.10334249585866928, 0.46046164631843567, -0.2067149579524994, -0.9739088416099548, -1.5288375616073608, 0.495045930147171, -2.359707832336426, 3.2363545894622803, 0.7067136168479919, -0.5664986371994019, -0.5063830614089966, 1.203994631767273, -0.844879150390625, -0.44862428307533264, -1.1653703451156616, 0.7339956760406494, 0.9360461235046387, 0.42173656821250916, -0.20870374143123627, -0.449767529964447, -1.279356837272644, -0.7827801704406738, -2.0884222984313965, -1.062211513519287, -0.2951807379722595, 1.1569408178329468, 1.709661602973938, 2.146202325820923, -0.10039655864238739, -0.4980183243751526, -0.4907749593257904, 1.5999103784561157, 1.9393409490585327, -0.18298546969890594, -0.10296246409416199, 0.9512609243392944, -0.9428897500038147, -2.171659231185913, -1.2059201002120972], [-0.011062477715313435, -0.7841206789016724, -1.0266019105911255, 0.09260106831789017, -1.2727251052856445, 1.456926941871643, -0.976777970790863, -0.24847407639026642, -1.3730404376983643, 1.0799365043640137, 1.6537429094314575, -0.6362154483795166, -0.2862051725387573, 1.2168742418289185, 1.3973597288131714, 0.9107926487922668, -0.9409236907958984, -0.7888593673706055, -0.984201192855835, 0.5426540374755859, 0.01113828457891941, -1.060987114906311, 1.2288151979446411, -2.4340121746063232, 0.46341589093208313, -1.076485276222229, -0.5244843363761902, -0.6664490103721619, 1.6613496541976929, -1.6806291341781616, -0.8922966718673706, 1.217193603515625, -0.42195361852645874, 0.6670491099357605, 1.1689355373382568, -0.3345303535461426, 0.12009274959564209, 1.136976957321167, -0.215431809425354, 0.05270027369260788, -0.6429115533828735, 0.6383408308029175, 1.1192736625671387, 0.008204100653529167, 0.9137793779373169, -1.020473837852478, -0.7513718605041504, -1.1362870931625366, 0.7164668440818787, 1.42843759059906, 0.4686957597732544, 0.15129412710666656, -0.18734554946422577, 0.08301109075546265, -0.9413179755210876, -0.3923948407173157, -0.5148261189460754, -0.5352188944816589, -1.0999374389648438, 0.4812586307525635, 0.8269728422164917, 0.14311553537845612, 2.6706297397613525, 1.0589226484298706], [-1.0157755613327026, -1.3459742069244385, -0.054285384714603424, 0.41325777769088745, 0.47855880856513977, -2.10296630859375, 0.7229671478271484, 0.37420785427093506, 0.22652290761470795, 1.1214206218719482, 0.504697859287262, -1.0793484449386597, 0.19349007308483124, -1.613405466079712, 0.36535391211509705, -0.12834808230400085, 0.328194797039032, 0.6277376413345337, -1.6235758066177368, 0.7374956011772156, -0.5162360072135925, 1.1448551416397095, -1.0967495441436768, 1.1323422193527222, -0.6992267370223999, 0.7099541425704956, -0.022410690784454346, -0.31845295429229736, -1.384436011314392, -0.8206668496131897, -0.8127269744873047, -0.7749282121658325, -0.8170716166496277, 0.5106935501098633, -1.0577729940414429, -0.3127473294734955, -1.824333667755127, -0.5999399423599243, 0.5326266884803772, 0.3079003691673279, -1.652338981628418, -0.8246626853942871, -1.7032508850097656, -1.050635814666748, 0.4396180212497711, 1.3421090841293335, -1.164071798324585, 0.9230782389640808, -0.6621811985969543, 1.2893903255462646, -1.1731783151626587, 1.3767551183700562, 0.4936848282814026, 0.24040041863918304, 0.30660027265548706, 0.10542947798967361, -1.1230084896087646, 0.9433709383010864, -0.5692316889762878, -0.487691193819046, -0.3959517478942871, -1.0505534410476685, -2.297417640686035, 1.1313954591751099], [0.20132778584957123, 0.2713053822517395, 1.0227296352386475, 0.7589854598045349, -0.3471319079399109, -0.9155307412147522, -0.5328895449638367, 1.6799894571304321, 1.3928059339523315, 0.03803684934973717, -0.033467795699834824, 0.3421178162097931, 0.28348207473754883, -0.1762877106666565, -0.04286564514040947, -1.2248320579528809, 0.14237667620182037, 0.34293800592422485, -0.7064568996429443, -1.2033350467681885, -0.22864098846912384, 2.4388184547424316, -0.9891409873962402, -0.2388814091682434, -1.847397804260254, 0.2261841595172882, 0.27204057574272156, -0.34935787320137024, 0.010991865769028664, 3.4399802684783936, 1.0316797494888306, -1.3993767499923706, 0.5337685942649841, -0.6437699198722839, 1.1870055198669434, -0.6721002459526062, -0.13276337087154388, 0.9592581391334534, -1.1262385845184326, -0.21295776963233948, -0.21430283784866333, 1.084276556968689, -0.27498725056648254, 1.7941137552261353, -1.0436224937438965, 0.21530508995056152, -0.945967435836792, 0.18681225180625916, -1.2696894407272339, 0.9474520087242126, 0.3987595736980438, -1.5645256042480469, 0.950954020023346, 1.3115788698196411, -0.705739438533783, 0.3627854287624359, -0.2674642503261566, -1.1316241025924683, 0.44691652059555054, -1.3392024040222168, -1.0618855953216553, -1.6951614618301392, 0.7773047685623169, 0.23119863867759705], [-1.505828619003296, -2.056321382522583, 0.5692967176437378, -1.2146365642547607, 0.5391329526901245, 0.21094074845314026, 0.6532756090164185, -0.16641655564308167, 0.9230809211730957, -2.3979854583740234, -0.1652468740940094, -0.4293161928653717, 0.020481746643781662, 0.8421075344085693, 0.8436170220375061, 0.8215944766998291, 0.6078663468360901, 1.3580007553100586, 0.19374653697013855, 1.0436375141143799, 0.6533488035202026, 0.40472841262817383, -0.7519093751907349, -0.14277061820030212, -0.6181049942970276, 1.4971823692321777, -1.5388813018798828, 0.17171235382556915, -0.3766153156757355, 0.23532216250896454, -0.5046067833900452, -0.8893753886222839, -0.15741446614265442, -0.1029801070690155, -0.5809445977210999, -1.1853013038635254, -1.441489577293396, -0.8095394372940063, -0.4590688943862915, 0.3218116760253906, -0.8179781436920166, -0.8968713879585266, -1.7342413663864136, -0.6719807982444763, 1.7512024641036987, 1.0927025079727173, -1.6177111864089966, -0.23855066299438477, 0.11910141259431839, 1.3855228424072266, 0.2632395029067993, 0.3353530466556549, -0.3244927227497101, 1.2015570402145386, -1.363264560699463, -0.46090540289878845, 0.35716211795806885, -1.026268482208252, -0.6901109218597412, -1.545027732849121, 0.4188244342803955, -0.020115375518798828, 0.7979727983474731, -0.2960725426673889], [-1.3338589668273926, 0.863822340965271, 0.7207117080688477, 0.2542136311531067, 1.4196614027023315, -1.6174298524856567, -1.3606008291244507, 0.10539592802524567, -1.960525393486023, -0.4443836808204651, 1.4494776725769043, -1.7739595174789429, 0.6889544725418091, 1.9094455242156982, -0.23672030866146088, -1.0722678899765015, 1.5895073413848877, -0.536018431186676, 2.183490753173828, 0.8706063628196716, -1.8834725618362427, 0.6234064102172852, -1.6727186441421509, 0.9536249041557312, -0.5020712614059448, 0.07242783904075623, -0.05201675742864609, 0.6477634310722351, -0.04132191836833954, 1.6517119407653809, -0.22123964130878448, 0.06619357317686081, -1.5570787191390991, 0.46907684206962585, -1.21172034740448, 0.0480661541223526, -0.204656720161438, 1.2948673963546753, -0.33079782128334045, -0.821450412273407, 1.0349422693252563, -1.7747852802276611, 0.002489612437784672, 0.06489549577236176, 1.1891223192214966, -1.7447575330734253, -0.3633395731449127, -0.006755325477570295, -0.9823882579803467, -0.4565870761871338, -2.0928359031677246, -0.19518223404884338, 1.0888373851776123, -0.966323733329773, -1.7021323442459106, -2.5846619606018066, -0.8777419924736023, -0.5138559341430664, 0.05640915408730507, -0.9308044910430908, 0.015029214322566986, -0.7167624235153198, 1.1883279085159302, 0.34484365582466125], [-0.14427131414413452, 0.4281954765319824, 1.0313842296600342, 1.0995399951934814, -0.2784726023674011, -0.3008701205253601, -0.5071417689323425, 1.184777855873108, -0.06732378154993057, 0.9192821979522705, 0.49637287855148315, -2.538774013519287, -0.513709306716919, -1.6597590446472168, -1.1907787322998047, 1.3242385387420654, 0.4284838140010834, 0.2030823975801468, -0.17122462391853333, 0.4366151988506317, 0.35033491253852844, -0.8317424058914185, -1.089043378829956, 0.07095472514629364, -0.2612922787666321, -0.2402770221233368, 0.6560817360877991, -2.166518449783325, 0.3470277488231659, 1.300794243812561, -1.2280150651931763, 1.5333720445632935, 0.2840372323989868, 0.02558353915810585, -0.4879986047744751, 1.5358673334121704, -0.1544255018234253, -0.613797664642334, 0.568281352519989, 1.2503293752670288, -1.236205816268921, 0.47427040338516235, -0.9766901135444641, -1.1984232664108276, 0.3059501349925995, 0.4800780415534973, -0.5974692106246948, 1.2002507448196411, -0.3777756094932556, -0.03413034975528717, -0.6783639788627625, -2.064849615097046, 0.9031345844268799, -2.3059799671173096, -1.0042378902435303, -0.6471885442733765, -0.8668780326843262, 0.4076068699359894, -0.39279282093048096, 1.1185994148254395, -0.4463871717453003, -1.0860497951507568, 1.4349207878112793, -0.33448803424835205], [-1.8946144580841064, 0.016416503116488457, 1.0258817672729492, -1.1942996978759766, -0.3405992090702057, 1.1705322265625, 0.45887792110443115, 1.420261025428772, 0.7069233655929565, -1.2121278047561646, 0.27569064497947693, -2.4610936641693115, -0.7524440884590149, -0.052073609083890915, 1.2760374546051025, -1.6627732515335083, -0.34969156980514526, -0.2208816409111023, -1.1566526889801025, 2.0925774574279785, -0.5639041066169739, -1.013340711593628, -0.5352775454521179, 0.5909217000007629, 1.018847107887268, -0.8726481795310974, 1.530163288116455, 1.5988895893096924, 1.1231251955032349, 0.43271368741989136, 0.009401952847838402, 0.513226330280304, 0.66451495885849, 0.22248680889606476, 0.8409407734870911, 0.5722282528877258, -0.6691312789916992, 0.04964068904519081, 1.4770125150680542, 0.11843866854906082, -0.9195000529289246, 1.1722155809402466, -0.09430725127458572, -0.9325881004333496, -0.15118053555488586, 1.4713695049285889, 2.249786376953125, 0.12019067257642746, 0.11645280569791794, -1.2696030139923096, 0.8388094902038574, -0.07291559129953384, 0.6355388760566711, -0.1552562564611435, 0.2544555962085724, 0.0789242759346962, 1.9488670825958252, -1.5945801734924316, -1.2042903900146484, -0.6203284859657288, 0.17834527790546417, -0.22604532539844513, -0.41015857458114624, 0.9309251308441162], [-0.05460789054632187, -0.21955835819244385, 1.1537222862243652, 2.1953299045562744, -1.4260460138320923, 0.7101359963417053, -2.796689748764038, 1.393405795097351, 1.8198052644729614, -1.131363868713379, -0.08085572719573975, 0.7112930417060852, 0.2646885812282562, 0.2493402361869812, 0.9017235040664673, -0.6814718246459961, 0.9111030697822571, 0.88767009973526, 0.49720004200935364, 0.24057339131832123, -0.3599846065044403, 1.2129426002502441, 2.3040759563446045, 0.3390972912311554, 0.7244102358818054, 1.9356589317321777, -0.10496463626623154, -0.4038081765174866, 0.49761736392974854, -0.2846870720386505, 0.1379123330116272, 1.3227920532226562, 0.4360351860523224, -1.2379382848739624, 0.6668266654014587, 1.0500131845474243, 0.8525663614273071, -0.9995918273925781, -0.326080322265625, -0.11338987201452255, -2.088859796524048, -1.2732688188552856, 1.5785857439041138, 0.2937629520893097, 1.7744885683059692, 1.0478551387786865, -0.6667283177375793, 1.066428780555725, 0.045834168791770935, 0.5733726024627686, 0.7615579962730408, -0.7881637811660767, -0.9948196411132812, -0.9104479551315308, 1.1762800216674805, -0.35481584072113037, 0.0908958911895752, -1.4988062381744385, 0.75360506772995, -0.8101924657821655, 0.6472945213317871, -0.4191817343235016, 0.994048535823822, 1.0847359895706177], [-0.2866981625556946, 0.9377965331077576, 0.1671569049358368, -1.281175136566162, -1.1224441528320312, 0.7540702819824219, -1.3864833116531372, -0.580064058303833, -1.5565800666809082, 0.1353408694267273, -1.5552177429199219, 0.31852081418037415, -0.1146094873547554, 0.3702564537525177, 0.5928157567977905, -0.8634839653968811, 0.8180496096611023, -0.731805682182312, 0.5576587319374084, -0.4058331847190857, 1.124463677406311, -0.17931967973709106, -0.5403923392295837, -0.059976495802402496, 0.443494588136673, 0.38049185276031494, -0.3059939444065094, 0.04250812157988548, -1.3171488046646118, 0.1930406242609024, -1.1117795705795288, 1.8568636178970337, 0.2686242163181305, -0.033912088721990585, -0.5148279666900635, 0.18169163167476654, 1.3543511629104614, 0.3741355836391449, 0.15932685136795044, 0.5321097373962402, 0.5410705208778381, 1.1241381168365479, -0.9623708128929138, -0.9144431948661804, 0.9266955852508545, 0.7573240995407104, -0.4269372820854187, 0.9216094017028809, 2.9204111099243164, -1.3414597511291504, -1.1590303182601929, -1.4059962034225464, 1.3189929723739624, 0.6532540321350098, 1.1570206880569458, -0.9319508075714111, -0.4766402542591095, 0.06665587425231934, 1.4517978429794312, 0.42294642329216003, -1.138427495956421, 0.9010571241378784, 0.8404373526573181, 0.154440775513649]], [[-0.31126755475997925, -1.633825659751892, -1.312991976737976, -0.3064867854118347, -0.6712002158164978, 0.3130292296409607, 1.398019552230835, -0.1605917066335678, 0.10965917259454727, 0.37676534056663513, -0.10623137652873993, -1.7100275754928589, -0.7369678020477295, 0.16082151234149933, 0.0062384526245296, 1.3226375579833984, -0.3906497061252594, 1.778928279876709, -1.8089698553085327, -0.6414799094200134, 0.4513486325740814, -0.04488536715507507, 0.6636821031570435, -0.1507962942123413, -0.48134130239486694, -0.708073616027832, -0.42193326354026794, 1.471052646636963, 1.073845624923706, 0.20087578892707825, -0.5821544528007507, -0.17455339431762695, -1.6029415130615234, 1.1085618734359741, 0.030785439535975456, 0.8106206655502319, -0.9862696528434753, -0.6860988140106201, -0.8885257840156555, 0.490283727645874, 0.05405588820576668, -0.34063178300857544, -0.6003024578094482, -0.16988934576511383, 0.5017129182815552, -1.2304010391235352, 0.02086555026471615, 1.108886480331421, 0.07600197941064835, -0.9753072261810303, 1.0087952613830566, -2.2167177200317383, 0.19018061459064484, -0.5259542465209961, -0.38011908531188965, 0.55415278673172, -1.5536835193634033, -0.08138252794742584, 1.020307183265686, -0.2878114879131317, 1.505366563796997, -1.5877320766448975, -0.08972834050655365, -1.1518056392669678], [0.699259340763092, 0.1458098441362381, -0.4377431869506836, 0.27056393027305603, 0.9945642352104187, -0.3267586827278137, 0.2447846233844757, -0.7891567945480347, -0.12771129608154297, 1.5143953561782837, -0.284902960062027, 0.12701737880706787, 0.46284541487693787, 0.15641839802265167, 1.1462332010269165, 0.001966227311640978, -1.337554693222046, -1.2519383430480957, 0.23224832117557526, -0.8310104012489319, -1.4724394083023071, -0.2650283873081207, 1.0401768684387207, 0.34567439556121826, -2.3058054447174072, -0.3958854675292969, 0.39053648710250854, 0.08969991654157639, -2.1597890853881836, 0.46926140785217285, 0.5970431566238403, 0.8529700040817261, 0.930584192276001, -0.7358877062797546, -0.4999230206012726, -0.8559557795524597, -0.17442204058170319, -0.1888599395751953, 0.1307845413684845, 0.30117231607437134, -1.1680485010147095, -1.0227863788604736, 0.5149991512298584, 1.0780085325241089, 0.1908871978521347, 0.43344417214393616, -0.025130445137619972, -0.6944199204444885, 2.2615485191345215, 1.2120351791381836, 2.6029815673828125, 0.6470953226089478, -1.3176614046096802, 0.7777363061904907, -0.7859163284301758, 0.5525839328765869, 0.6112077832221985, -0.6710413694381714, 0.7625138759613037, 0.29259854555130005, 0.5015050172805786, 0.03896922990679741, 2.3291499614715576, -0.1390099823474884], [-1.0944712162017822, -0.2605668604373932, 0.31403642892837524, 0.7506927847862244, -0.19885389506816864, -1.3111640214920044, 0.24171580374240875, -0.048057712614536285, 0.3525422215461731, 0.2993653118610382, 1.2221962213516235, 0.8022392988204956, 0.8032981157302856, 0.133571594953537, -1.7289382219314575, -0.0072916592471301556, 0.2354741096496582, -1.582431435585022, -1.043169379234314, -0.50648432970047, -0.8721634149551392, 0.3299238085746765, 0.28020766377449036, 0.27363041043281555, -1.2591955661773682, 1.0985912084579468, -0.6193689107894897, -0.4166891276836395, -0.6791438460350037, -0.15335498750209808, -0.5933610796928406, 0.025438042357563972, -0.5932837724685669, 1.1002768278121948, -2.36734676361084, 0.44871264696121216, 1.1164554357528687, -0.10735362023115158, -0.6461482048034668, -1.810980200767517, 0.41294530034065247, -0.6039702892303467, 1.2623668909072876, 0.4646873474121094, -0.9801568984985352, 0.697616696357727, 0.5839145183563232, -0.45476871728897095, 2.205500841140747, 1.8334287405014038, -2.2093124389648438, 0.02921815775334835, -1.0932695865631104, -0.012302037328481674, 0.8345724940299988, 1.2677891254425049, -1.4890493154525757, 1.5859774351119995, 0.6061033606529236, -3.7490406036376953, 0.6888012290000916, 0.6728232502937317, 1.6374350786209106, 0.6904560327529907], [-1.815544843673706, 0.1558510810136795, 1.5179327726364136, -1.3619842529296875, 2.0486361980438232, -0.09868750721216202, 0.5933120250701904, 0.7429192662239075, -0.6801642775535583, -3.0283524990081787, -0.8458359241485596, -0.5416452288627625, -0.47029468417167664, 0.2813439965248108, -0.7280969619750977, -0.7657070755958557, 1.1010452508926392, -0.505820095539093, -0.7345246076583862, 0.5798590183258057, 0.07269231975078583, -0.42672184109687805, 0.3545769453048706, 1.3417786359786987, -1.9743469953536987, -0.5595939755439758, -0.03005685843527317, 0.45909175276756287, 0.15215323865413666, 1.0407580137252808, 1.6737910509109497, 0.8221412301063538, -0.5936598181724548, -0.09206240624189377, 1.3759515285491943, 0.15144644677639008, -1.2671422958374023, -0.47585880756378174, 1.447153925895691, 1.74116051197052, 0.7161998748779297, 0.28574076294898987, 0.10595621913671494, 0.7877230644226074, -0.5909456014633179, 0.004836553242057562, 0.8606547713279724, -1.929835319519043, 0.12608186900615692, -0.07076389342546463, 0.3359331786632538, 0.14325641095638275, 1.9586782455444336, -1.6403746604919434, 0.6990482211112976, 0.4012935757637024, -0.1595035195350647, -0.6368409991264343, -1.8567430973052979, -0.4640183746814728, 0.542281448841095, -0.6475242376327515, -0.7182794213294983, 1.2683134078979492], [0.5535072684288025, 0.28800031542778015, 0.48885267972946167, -0.6195751428604126, 0.6402856707572937, -0.3676314949989319, -1.907313346862793, -0.3816147446632385, 1.1331347227096558, 1.9562530517578125, -0.7539899945259094, -0.8243610262870789, 1.7262451648712158, -0.3073180317878723, 1.352034091949463, -0.03684188425540924, -0.20406125485897064, -0.012875836342573166, -0.8071743845939636, -1.8698890209197998, 0.4884239137172699, 0.5237534046173096, -0.502406895160675, 0.16870561242103577, 1.4422905445098877, -0.2942466139793396, 0.06107575446367264, 1.8155235052108765, -0.16931036114692688, 1.7104347944259644, 0.6318448185920715, -1.1047217845916748, -0.9108427166938782, 0.7697288393974304, -2.500281572341919, 1.528267741203308, -0.9184139370918274, -0.7492311596870422, 0.9036695957183838, 0.5977103114128113, 2.074956178665161, -0.579376220703125, 0.046277280896902084, 0.5725021958351135, 0.13288578391075134, -1.3721132278442383, -0.7021996378898621, 0.8208670020103455, -0.9493730068206787, 0.04545307531952858, -0.25722986459732056, -1.4086544513702393, 0.10423202067613602, 0.38345423340797424, -2.576920747756958, -0.5923125743865967, 1.1181156635284424, -0.8926628828048706, 1.474948763847351, 1.2727060317993164, -1.9143880605697632, -0.5259888172149658, -0.16595353186130524, -1.2183376550674438], [1.8236336708068848, 0.2444811910390854, -0.7244527339935303, -1.1772385835647583, -0.43615174293518066, -0.4460369050502777, -0.8778502345085144, -0.7656209468841553, 2.115889310836792, 0.12138540297746658, -0.7652015686035156, 0.9164065718650818, 1.0412912368774414, 0.026010382920503616, -0.3168112337589264, -0.4500276744365692, -0.8047758340835571, 1.0879764556884766, 2.848478317260742, -1.8502001762390137, -0.26037007570266724, 0.9001337289810181, -0.7495353817939758, -0.9005469083786011, -1.7853453159332275, -2.1280264854431152, 1.3059672117233276, 1.1773265600204468, -0.027085741981863976, 1.6865296363830566, 0.2355867326259613, 2.3119735717773438, 1.1848127841949463, -0.7597147822380066, -0.17614173889160156, 0.2536540627479553, 2.068742513656616, -1.331412672996521, -0.18855495750904083, -0.9066638350486755, -0.7669628262519836, 1.8028295040130615, 2.5876100063323975, 0.10221115499734879, -1.0634839534759521, -1.4806050062179565, 1.598171353340149, 1.2552459239959717, -0.43421098589897156, -0.0403442308306694, -0.3647333085536957, 1.612173080444336, 0.0111728236079216, 0.2640262544155121, -0.28455841541290283, -1.1346415281295776, -0.6331868767738342, -0.2610613703727722, -0.5579301714897156, -0.008703036233782768, -1.3479896783828735, 0.9955010414123535, -1.386734962463379, 0.010809101164340973], [0.46750956773757935, 1.3877149820327759, -2.000490427017212, -0.5948857665061951, 1.91456937789917, 0.5921320915222168, -0.097413070499897, -0.7278419733047485, 0.7108716368675232, 1.6490446329116821, -0.07254356890916824, -0.8014733791351318, -0.18582895398139954, -2.046112298965454, -0.9556812047958374, -0.28377246856689453, -0.31295594573020935, 0.6547421216964722, 0.6184942126274109, -0.7300668954849243, 1.6041462421417236, -0.12017872929573059, -0.7485383749008179, 0.054123010486364365, -0.837454617023468, -1.6279027462005615, -1.1326392889022827, -0.11989687383174896, 1.0748822689056396, -2.058478832244873, -0.6260658502578735, 0.6669113636016846, -0.7024955749511719, -0.04049283266067505, 0.13573943078517914, 1.0023761987686157, 0.11143721640110016, -0.1760084182024002, 2.2418813705444336, -0.6301291584968567, 0.2702856659889221, -3.3652865886688232, 0.0026107411831617355, -2.387458086013794, -0.47846370935440063, -2.947700262069702, -0.15035167336463928, 1.0192054510116577, -0.4044959843158722, -0.16882352530956268, 1.2531366348266602, -0.561224102973938, 0.3298380672931671, -1.4393746852874756, -0.9334384202957153, -2.357325792312622, 0.40810754895210266, -1.041817307472229, 0.10881348699331284, 0.4216797947883606, 0.435268759727478, -0.6838092803955078, 0.30300694704055786, -0.18322736024856567], [-0.9655721783638, -1.590694546699524, -0.028370872139930725, -0.22586044669151306, 0.00943665113300085, -0.0074063437059521675, 0.12288948148488998, 0.8184366822242737, 1.4742602109909058, -0.41458648443222046, -0.6745707988739014, 0.5878909826278687, 0.08446675539016724, -0.35319826006889343, -0.03852174058556557, -0.7199504375457764, 0.7827072739601135, -0.5524682402610779, 1.2341996431350708, -0.9681223034858704, -0.19985130429267883, 0.8093665838241577, 0.8124493360519409, 0.09969866275787354, 0.03921542689204216, 1.3329675197601318, -0.9282236099243164, -2.5573458671569824, 1.3629436492919922, -0.5731582045555115, -0.8185078501701355, -0.014836634509265423, 0.8095337152481079, 0.028647564351558685, -0.10401997715234756, 0.15486203134059906, 1.5734350681304932, -0.007412651088088751, 0.8269698023796082, 0.8285654783248901, 0.7029123306274414, -0.32353395223617554, -0.8679145574569702, 1.6798771619796753, 1.7559894323349, 0.3464139401912689, -1.4221758842468262, 0.5222637057304382, 0.6271716952323914, 0.08894725888967514, -1.1883749961853027, 0.20364879071712494, -0.9378261566162109, -0.5856233239173889, 0.37563827633857727, -0.2472139298915863, 0.4682615101337433, -0.47030961513519287, 0.21322070062160492, 0.10997350513935089, 0.0862070843577385, 1.482102870941162, 0.9574704766273499, 0.2869165539741516], [0.9291006326675415, 2.301206350326538, -0.7692360281944275, 1.4729797840118408, -0.9485248923301697, -2.5273308753967285, -0.5877767205238342, 2.14471173286438, 1.0879539251327515, 1.025308609008789, -1.500324010848999, 3.1464920043945312, -0.33032867312431335, -0.2794433832168579, 1.7371057271957397, 0.2191193550825119, -1.5567030906677246, 0.17291763424873352, -0.4505856931209564, 0.5966894030570984, 0.1822904795408249, 0.44663766026496887, -0.7675304412841797, -2.368989944458008, -0.8399706482887268, -1.5619657039642334, -0.3307429254055023, -0.04488462954759598, -1.80880606174469, -0.5602694153785706, 1.6523995399475098, 0.7262890338897705, 0.7160658836364746, 0.4052943289279938, -0.8775001168251038, 2.1110713481903076, -1.0928537845611572, -0.1426444798707962, -0.33683153986930847, 0.2716589868068695, -1.3535643815994263, -1.4780998229980469, -2.216350793838501, -0.6707425713539124, -1.5930782556533813, -0.9200772047042847, 0.657149612903595, -0.8489448428153992, -0.010087192058563232, 0.22927263379096985, 0.5671122670173645, -0.7254976630210876, -0.11600062251091003, 0.0818478986620903, -1.6501520872116089, -1.0937612056732178, 0.7662957906723022, -0.5974608659744263, -0.2283601313829422, -0.1405850499868393, -1.4171398878097534, -0.05122203752398491, -0.7755612134933472, 0.9980236291885376], [1.5692973136901855, 1.3887523412704468, 1.1607857942581177, 0.781565248966217, 0.06970545649528503, 0.09331757575273514, 1.1436747312545776, -0.3316820561885834, 1.378421425819397, -0.06500054895877838, -1.085910677909851, -0.7342050075531006, -1.3485933542251587, 1.74961256980896, 0.5393688678741455, 0.5913384556770325, -0.2823488414287567, -1.527956485748291, -1.0251696109771729, 0.4379808008670807, -1.4372261762619019, -1.3172143697738647, 0.9675083756446838, -0.8570849299430847, 1.0858144760131836, -0.9099048376083374, -0.7404391169548035, -0.6306585073471069, 1.4338361024856567, 1.8775969743728638, 0.28927507996559143, 0.8441290259361267, -1.1951406002044678, -0.560271143913269, -0.17422239482402802, -1.155376672744751, -0.20626911520957947, 1.1461769342422485, 0.7745732665061951, -0.1760801374912262, -1.1676619052886963, 0.23065021634101868, 0.6584824323654175, 1.324562668800354, -0.009877345524728298, -0.35276198387145996, 0.333858847618103, 1.6290562152862549, -1.2063244581222534, 0.07120299339294434, 1.7576583623886108, 0.8372432589530945, -1.6816647052764893, -2.4429633617401123, 0.9923718571662903, -0.1392795890569687, 0.35784032940864563, 0.864284336566925, 1.2651978731155396, 0.4169448912143707, -0.736356794834137, 0.014938847161829472, 1.2742940187454224, -1.479832649230957], [-1.5120675563812256, 0.23943892121315002, 0.4824756979942322, -0.6477785110473633, 1.4282217025756836, 0.3094874620437622, 1.00900137424469, -0.1410733014345169, 0.5262024402618408, -0.5146346092224121, -0.6862275004386902, -1.1825922727584839, -0.11405078321695328, 0.4842548072338104, 0.6807953119277954, -0.24633102118968964, -1.9999308586120605, 0.8394330143928528, 1.1693825721740723, 0.33260104060173035, 1.5693304538726807, 1.5369880199432373, -1.2892580032348633, 1.6853958368301392, -0.34591054916381836, -0.5281897187232971, -0.6560148000717163, -0.3032620847225189, 0.4496411681175232, 0.05016026645898819, 1.7226959466934204, 0.13873934745788574, 1.4046608209609985, -1.4269105195999146, -0.7834709286689758, 0.21880877017974854, -0.2727666199207306, 1.890929102897644, -1.4038357734680176, -0.6597940325737, 0.7621568441390991, 0.44876790046691895, -0.6442002654075623, -1.109034538269043, 0.12346245348453522, -0.10091224312782288, 1.1213610172271729, -1.0764154195785522, -1.0310660600662231, 1.2066861391067505, -0.6615270972251892, 1.533352255821228, -1.0354357957839966, 0.26033535599708557, 0.5908212065696716, -1.1993037462234497, 0.30921870470046997, 0.16926367580890656, 0.29962965846061707, -0.719188928604126, -0.5982891917228699, 0.20739148557186127, -1.094519853591919, -0.32133734226226807], [-1.0162408351898193, 0.3889338970184326, 0.4402914047241211, 0.7248733639717102, 0.43394312262535095, 0.2405143529176712, 2.0858137607574463, 2.72709584236145, 1.053209900856018, 0.4582694470882416, 0.951450526714325, 0.019158393144607544, -1.153706431388855, -0.6382384300231934, 0.6011850237846375, -1.058443307876587, 0.8754989504814148, -0.9247841835021973, -0.8746116161346436, -0.8470256328582764, -0.10827770084142685, -1.2436519861221313, 1.8226133584976196, -0.7801300287246704, 0.8913984298706055, 0.3572287857532501, 0.031485751271247864, 0.5445228219032288, 1.0412896871566772, -0.47616147994995117, -0.4928244650363922, 1.054093837738037, -0.3381257951259613, -0.5881092548370361, -0.7155128717422485, 1.1397370100021362, 1.626592993736267, 1.0090410709381104, -0.40096673369407654, 2.1137709617614746, -1.2331072092056274, 0.9709633588790894, -0.7204899191856384, -0.7832475900650024, 0.9775804877281189, -1.2330280542373657, 0.08374620229005814, 0.5345967411994934, 0.3732927739620209, 0.631066620349884, 0.48999106884002686, -0.35587623715400696, -0.16379621624946594, -1.5446091890335083, -0.5856989622116089, 0.024182746186852455, -1.3868404626846313, 0.11712290346622467, -0.37938588857650757, 0.2557627856731415, 0.39315131306648254, -1.639596939086914, -0.18939058482646942, 1.2990870475769043], [-0.44422706961631775, -0.42029938101768494, -0.02605525031685829, -0.08726196736097336, 1.2723166942596436, 1.758579969406128, -1.229793667793274, -1.0023173093795776, 1.9679933786392212, -0.24353191256523132, 0.6416472792625427, 0.12230533361434937, 0.24876543879508972, 0.7575832009315491, -1.3405390977859497, 0.04046347364783287, 1.327630639076233, -0.17602911591529846, 0.025003086775541306, 0.5300042033195496, -0.47122061252593994, 1.3536044359207153, 0.31436851620674133, 1.0328766107559204, -1.006130337715149, -0.20612601935863495, -1.142632246017456, -1.2716771364212036, 1.6150591373443604, -0.0684816837310791, -1.4400055408477783, -0.6923975944519043, -0.6617415547370911, -1.1136634349822998, 0.08763064444065094, 0.8639155030250549, 1.3053600788116455, -0.8667987585067749, 0.5344215035438538, 1.4746246337890625, 0.8614041209220886, 0.042319390922784805, -0.8988246917724609, 2.058741331100464, -0.1671382188796997, 0.4229089915752411, -0.7846723794937134, 0.5170933604240417, -0.4501166343688965, -0.8278093338012695, -0.051614757627248764, -0.007211350370198488, 2.1410012245178223, -0.07486838847398758, -1.251551628112793, 1.0156925916671753, 0.36999398469924927, -0.10908389091491699, -0.7778695225715637, 0.6304951906204224, -0.3586026728153229, 0.6077878475189209, 0.3705030381679535, 0.6492236852645874], [-0.23660719394683838, 3.141127824783325, -0.09082786738872528, -1.5457547903060913, 0.3115004897117615, 0.4611653685569763, -0.36391887068748474, 0.5727981328964233, -0.3193208873271942, 0.36772775650024414, 1.1316618919372559, -0.16728536784648895, 1.0109219551086426, 0.8475689888000488, -2.3331267833709717, -0.9326637983322144, -0.5152154564857483, -0.05297510325908661, -0.18972371518611908, -0.19330133497714996, -1.2524712085723877, 1.1590523719787598, -0.536984920501709, -0.17953503131866455, 0.9447252154350281, -1.0890557765960693, -1.4330288171768188, 0.7053103446960449, -0.25481224060058594, -1.659636378288269, 0.34304943680763245, -0.41573700308799744, -0.3922463059425354, 1.391624927520752, 0.28503915667533875, 0.02492232620716095, 0.5936241149902344, -0.8948531150817871, 0.38787856698036194, -0.01852712780237198, -0.1587294191122055, -0.05192289128899574, -0.8909884095191956, -0.808718740940094, 1.314632773399353, 1.9673285484313965, -1.058344841003418, -1.599233627319336, 0.8708231449127197, -0.4685148000717163, 0.17656926810741425, 0.6088211536407471, 0.6057928204536438, 0.26236310601234436, 1.3032386302947998, 0.6696351170539856, 1.1931750774383545, -0.3553939461708069, 0.27188268303871155, 0.19292917847633362, 1.2637170553207397, 0.27978986501693726, 0.886006772518158, -0.03933180123567581], [0.46787965297698975, 0.6482694745063782, -0.1528944969177246, -1.2093150615692139, 0.812518835067749, -1.0625450611114502, -0.12214308232069016, -1.2209341526031494, 1.445733904838562, 0.6021465063095093, 0.3635264039039612, -1.3230795860290527, -0.7512364983558655, 0.1964588761329651, 0.33928728103637695, 0.9349398612976074, 0.31222954392433167, 0.06774821877479553, -1.0907374620437622, -0.25866055488586426, -0.9080476760864258, 1.044511079788208, -0.5131070017814636, 1.222408413887024, -0.2517062723636627, -0.02987331710755825, 1.541212558746338, -1.5747148990631104, 0.9313893914222717, 0.2193046361207962, -1.3352508544921875, 0.6001631617546082, -0.10297916829586029, -1.6550768613815308, 0.4531874358654022, 1.3953533172607422, 0.9087499976158142, -2.304154634475708, -0.5796934366226196, 0.3081700801849365, 0.12562718987464905, -0.18706925213336945, 0.47169211506843567, -0.6766178607940674, 1.7425824403762817, -0.14881569147109985, -0.7449934482574463, -0.7953739166259766, -1.7154840230941772, -0.10622920840978622, 0.6843132972717285, 1.9290046691894531, -0.515913188457489, -1.188791275024414, 1.029227375984192, 1.070327877998352, -0.5244888663291931, -0.3278094530105591, 1.8233851194381714, 0.7079766988754272, 1.259814739227295, 0.3585486114025116, 0.882588267326355, 0.30328187346458435], [-1.0095123052597046, 1.4167360067367554, 1.8091158866882324, -0.2805015742778778, 1.049286961555481, 0.4196382761001587, 0.33791986107826233, -0.22057418525218964, 0.9744083285331726, 0.3190160393714905, -1.3444744348526, 0.7043449878692627, 0.21531063318252563, 1.170248031616211, 0.5260518193244934, 0.07851198315620422, -0.3082179129123688, 0.5473707914352417, -0.3437192142009735, 0.7718632221221924, -0.4868753254413605, -0.18745729327201843, 0.5902910828590393, 0.6724627017974854, -0.6541134715080261, 0.4615098536014557, 0.4575044810771942, -1.0400526523590088, 0.17734038829803467, 1.4315307140350342, -1.9962953329086304, -0.25276196002960205, 0.4148028492927551, 1.4788841009140015, -0.5913833379745483, 0.08921768516302109, -0.9738837480545044, -0.5439590215682983, 1.2570074796676636, -2.2818291187286377, -0.3050744831562042, 0.8791813850402832, -0.5866638422012329, -0.19729888439178467, -0.0025796727277338505, -0.23456472158432007, 0.9106429219245911, 1.8310261964797974, -0.8737010955810547, 0.827620804309845, -0.8100365996360779, -0.498666912317276, 2.3964929580688477, 1.0128799676895142, 1.2431137561798096, -2.180820941925049, -0.7792257070541382, 1.0165270566940308, -1.798916220664978, 1.3689326047897339, -0.9412240386009216, -0.6501768827438354, -0.8201509118080139, -0.7918254733085632]]]], "dtype": "float32"}, {"__nd__": [[[[-0.877932608127594, 0.47240909934043884, -0.43540412187576294, -1.7018442153930664, 1.1300969123840332, 0.07020007818937302, -1.1697602272033691, 0.4740818738937378, 0.45807021856307983, -1.0432578325271606, -0.23126573860645294, 0.46279069781303406, 1.1446919441223145, -1.2687050104141235, 0.8636118173599243, -0.35739755630493164, 0.7664951682090759, -1.1347678899765015, -0.4720315635204315, 0.0037250560708343983, -1.7793446779251099, -0.8886452913284302, -1.666334867477417, -0.8401347994804382, 0.29984617233276367, -1.0038272142410278, -1.0832879543304443, 1.220294713973999, 0.6427050232887268, 1.1213583946228027, -0.334852397441864, -0.47384291887283325, -0.9877446889877319, 1.6843427419662476, -1.7659050226211548, 0.6863232851028442, 0.8446780443191528, -1.0150188207626343, -1.3655866384506226, -0.6529760956764221, -0.693239688873291, 0.882468044757843, -3.1800570487976074, 0.6477404236793518, -0.810769259929657, 0.3443303108215332, -0.8008763194084167, 1.5504661798477173, 0.9145922064781189, -0.603032112121582, 1.6908613443374634, -0.25688984990119934, 0.5522445440292358, -0.458251029253006, -1.290876030921936, 0.4970652163028717, -1.4433536529541016, -0.7611109614372253, -0.13361182808876038, -1.1167268753051758, 0.016046255826950073, -0.11946887522935867, 2.40818452835083, 0.8151846528053284], [0.4139866828918457, 0.03797227889299393, -0.34893354773521423, 1.204884648323059, -1.143868327140808, 0.4905110001564026, -0.9972974061965942, 0.2492755800485611, 0.8328572511672974, 1.0767260789871216, -1.151818871498108, 0.4166393280029297, 0.09555280208587646, 1.402963399887085, 0.3338741660118103, -2.378502368927002, 1.2362303733825684, 0.7003299593925476, 0.19292502105236053, -0.056041326373815536, 0.16907168924808502, -0.56770920753479, -1.0123486518859863, 0.3456563651561737, -0.3003554940223694, -0.6252076625823975, 0.6008721590042114, -0.8476753830909729, -0.25566932559013367, -0.2949274182319641, 1.2382862567901611, 0.20269733667373657, -0.07283852249383926, 0.701026976108551, -0.49490347504615784, -1.9698309898376465, -0.46999314427375793, -0.18184205889701843, -1.3595635890960693, 0.5579313039779663, 0.7192986607551575, -0.37854665517807007, -0.933048665523529, 0.6147521734237671, 0.7705161571502686, -0.947089672088623, 0.5776655077934265, 0.5926478505134583, 1.1320033073425293, 0.7757192850112915, -0.4408031404018402, -0.4231683313846588, 0.04221402481198311, 0.3741784989833832, 0.38881754875183105, -0.7525570392608643, 1.4706588983535767, -0.4882391393184662, 0.8607191443443298, 0.1301112323999405, -1.3042958974838257, -0.5862047076225281, 0.33932170271873474, 0.3385559022426605], [-0.13398973643779755, -0.640155553817749, 1.7998591661453247, -1.293237328529358, -1.3043603897094727, 0.32426169514656067, 1.2104986906051636, 0.6073974967002869, -0.8421857357025146, -0.5363501310348511, 0.5548150539398193, -0.8824484348297119, -0.2634337246417999, -1.5742542743682861, -0.5891746282577515, -0.577958345413208, -1.037896990776062, -0.3446315824985504, -0.616769015789032, -0.3681032955646515, -1.4803804159164429, -0.5964298844337463, -1.4488542079925537, -0.11087022721767426, -0.2901117205619812, 0.431017130613327, -0.9732162356376648, 0.0014271429972723126, -0.00564511539414525, -1.4349414110183716, 0.5098928213119507, 0.3987478017807007, -0.4638347327709198, -0.44069844484329224, 0.5763387680053711, 0.3467826545238495, 0.8162006139755249, -1.0058709383010864, 0.9920852184295654, 0.9150201678276062, -1.6507729291915894, -0.21343903243541718, 0.7569667100906372, 0.01868770644068718, 0.5120618939399719, -1.1329188346862793, -0.08241133391857147, 0.3617171347141266, -0.9630151987075806, -0.5764431357383728, 0.4404814541339874, -0.8092737793922424, -1.157829761505127, 0.27123522758483887, -0.2675967812538147, 0.8331036567687988, -0.4204481244087219, 0.2472071647644043, -1.3565466403961182, 0.5562812089920044, -0.19865259528160095, -0.010018251836299896, -1.2215982675552368, 1.2351117134094238], [-0.5267093181610107, -0.7078009247779846, 1.0423554182052612, -1.2256994247436523, -0.9368893504142761, -0.14663659036159515, 0.6964132189750671, 0.5707158446311951, -1.282088041305542, 0.08100827038288116, 0.9598376750946045, 1.011986494064331, -1.0966252088546753, -0.18637719750404358, -0.2549550235271454, -0.5524161458015442, -0.3386031985282898, -0.2779443562030792, -1.889654517173767, 0.9466730356216431, 0.6699289679527283, -0.524344265460968, -1.1710209846496582, 1.0649805068969727, 0.3192351162433624, 2.932912826538086, 0.5224400758743286, -2.0618951320648193, -0.0009244779357686639, -1.2588226795196533, -1.429254174232483, 0.2111915647983551, -0.24404536187648773, 0.16839946806430817, 0.44719773530960083, 0.856335461139679, -0.27838051319122314, -0.1914598047733307, 2.028204917907715, -1.8336440324783325, 1.507688283920288, 0.06772259622812271, -0.13198064267635345, 0.8505218029022217, 1.0350509881973267, -0.248270183801651, -0.9826463460922241, 0.6945169568061829, -0.4171833395957947, -1.0608994960784912, -0.8812442421913147, -1.570694088935852, -1.2377923727035522, 0.5202172994613647, -1.073571801185608, 1.5851249694824219, 0.04243814945220947, -0.5040790438652039, -0.08470635861158371, -1.0330778360366821, 1.694214940071106, 0.4869946539402008, 2.978924036026001, 0.7115088701248169], [0.05084021016955376, -2.272132158279419, 1.1913199424743652, 0.3705199360847473, -0.42473122477531433, -0.11879249662160873, -0.5779892802238464, 0.9936654567718506, 0.7599894404411316, -0.02749255858361721, 1.8995540142059326, -0.5835047364234924, -0.01983977109193802, -0.7393052577972412, -0.6746383309364319, 0.011899692937731743, -1.889062762260437, -1.8522343635559082, 2.189423084259033, -1.1967439651489258, 1.0925650596618652, -1.3555043935775757, 1.0032155513763428, 1.556401252746582, 3.2633142471313477, -2.486530303955078, -1.3849937915802002, 1.159729242324829, 0.7257177829742432, -1.693372368812561, 0.46977269649505615, -0.8644357323646545, -0.8857221007347107, -0.750678300857544, -0.874980092048645, 1.4639207124710083, 0.02739631198346615, -1.2777475118637085, -0.23971156775951385, -0.6478890180587769, 0.48101291060447693, 2.713981866836548, -0.1824345588684082, -1.899751901626587, -0.1322954297065735, -0.1680113524198532, 0.41789403557777405, -1.4001643657684326, -0.7957239747047424, -0.11329580843448639, -1.4424644708633423, -1.2004499435424805, -1.6053621768951416, 0.5822890996932983, 0.7384276390075684, 0.7353358268737793, -0.7409637570381165, -0.5963114500045776, 1.079367995262146, -0.2022385597229004, 0.24239540100097656, 0.1260100156068802, -0.35372394323349, -0.6941772103309631], [0.9532274603843689, 1.2420779466629028, 0.4137590527534485, 0.11531054228544235, -0.5778430700302124, -0.30837443470954895, -1.0434520244598389, 0.4900355041027069, 0.13951964676380157, 0.5168839693069458, -1.3279308080673218, 2.8679018020629883, 0.15247981250286102, -0.3568875789642334, -1.3763766288757324, -0.8816028833389282, 0.6886472105979919, 0.9311915040016174, 1.905781626701355, -1.0373287200927734, -0.610984742641449, -1.9536832571029663, -0.03771331161260605, 0.744036078453064, 2.0003092288970947, 1.5813497304916382, 0.6873915791511536, -0.15233156085014343, 0.5810502767562866, 0.8293691873550415, -0.9568465948104858, 1.6999030113220215, -0.7981995344161987, 0.9539733529090881, -1.5402547121047974, 0.11732490360736847, -0.12875865399837494, 0.40582743287086487, 1.080601453781128, -0.8009663820266724, 0.10673577338457108, 0.9833114147186279, -0.41684120893478394, 0.631583034992218, 0.1909143626689911, 2.391181707382202, 0.9483626484870911, 1.2692878246307373, -1.435640811920166, 0.5141074657440186, 0.8445183038711548, 1.359061598777771, 1.6400200128555298, -0.890222430229187, -0.10685694962739944, -1.0480996370315552, -0.16808128356933594, 0.7719356417655945, -0.3689377009868622, 0.049838680773973465, 0.539313793182373, -0.6375845074653625, -1.638596773147583, -0.4272097051143646], [0.9171552062034607, 0.857119083404541, -1.304264783859253, -0.24115271866321564, 1.5644505023956299, 0.8023303151130676, -0.6129562854766846, 1.3856042623519897, 0.29540401697158813, -0.19185870885849, -0.18503950536251068, -0.10298753529787064, 1.2846970558166504, -0.2761589586734772, -0.8017401099205017, 0.15132665634155273, 0.15736563503742218, 1.4424694776535034, 0.8138443231582642, -0.29045289754867554, -1.3969260454177856, 1.4543907642364502, 0.3929669260978699, 0.5553497076034546, 0.24555636942386627, -0.1278192698955536, -0.024278860539197922, 2.4757039546966553, 1.8093069791793823, 0.08187524229288101, 0.2629242241382599, -1.0765303373336792, 1.310573697090149, 1.3621747493743896, -0.0572977289557457, -2.1062209606170654, 2.3098888397216797, -0.16319186985492706, -0.6011064052581787, -0.8572262525558472, 0.4207751154899597, -1.102827548980713, 0.3822548985481262, 0.9153040051460266, 1.1186895370483398, -0.9251834154129028, 0.19065062701702118, 0.5353222489356995, 0.2545355260372162, -0.2032010555267334, -0.9796038866043091, -0.5457821488380432, 0.9838692545890808, 0.9398505687713623, -0.9641216993331909, -0.08544593304395676, -0.5657476782798767, 1.4193490743637085, -0.9623637795448303, 1.591849446296692, 0.5846052169799805, -1.229514479637146, -0.707206666469574, -0.19419220089912415], [-0.11173807084560394, 1.920534372329712, 0.2891436219215393, -0.26041796803474426, -0.05655699595808983, -1.054498553276062, -0.1574007272720337, 0.9150384664535522, -0.15644364058971405, 1.1315900087356567, -0.11114402860403061, -0.3826734125614166, -2.242567539215088, -1.5342994928359985, -0.23245121538639069, -1.3690372705459595, -0.36024144291877747, 0.2842368483543396, -1.0827070474624634, 0.19166287779808044, -0.08590836822986603, 0.16071811318397522, -1.529322624206543, -1.1165393590927124, -0.57552570104599, -0.6443792581558228, -0.7468743920326233, -2.4556190967559814, 1.8374167680740356, 0.420602411031723, 1.933780312538147, 0.8071696758270264, -1.4668347835540771, -0.015858709812164307, -1.0628554821014404, 0.4456974267959595, -1.2364808320999146, -0.6318315267562866, 0.7444415092468262, -0.26767969131469727, 0.31129324436187744, 1.2699673175811768, -1.1798003911972046, 0.3462001085281372, 0.16920509934425354, 0.6797260642051697, -1.4648903608322144, -0.025113599374890327, -0.23538561165332794, 0.3768364191055298, 1.5506131649017334, -0.6507202982902527, -3.0014162063598633, -0.9893770813941956, -0.9474010467529297, 0.7828783392906189, 0.7488701939582825, -0.7908009886741638, 0.15852950513362885, -0.16736850142478943, -0.3672586977481842, 1.5489165782928467, -0.9818601012229919, 0.6349430680274963], [0.3961876332759857, 0.6317539215087891, -1.1299636363983154, -1.0146865844726562, 0.2574392855167389, 0.29820457100868225, -0.4148772954940796, -1.000431776046753, -1.003609538078308, -0.9055464267730713, -1.5484644174575806, 0.4681127965450287, 0.27507123351097107, 0.5732461214065552, -0.4075060784816742, 0.3656895160675049, -0.9629161357879639, -1.2734744548797607, 1.2254087924957275, -1.012712001800537, 0.4975428283214569, 0.8093356490135193, 0.5505187511444092, -0.22111521661281586, 0.72984778881073, 1.5254881381988525, 1.7853734493255615, 0.2831525504589081, 0.27369362115859985, 1.3566128015518188, -0.7355672717094421, -0.12412422895431519, -1.6273462772369385, 2.7442469596862793, 1.948844075202942, 1.700771450996399, -0.17314735054969788, 0.3378615379333496, -0.9244936108589172, -1.1905944347381592, -0.19960643351078033, 1.3969155550003052, 1.6423637866973877, -0.822717010974884, 0.29857146739959717, 0.7034040093421936, 0.2592802047729492, -0.013944385573267937, 0.263422429561615, -1.2312486171722412, -0.3403494358062744, -1.2422749996185303, 1.556087613105774, -0.9270965456962585, 0.529725968837738, -0.044798530638217926, 0.6442126631736755, -0.2841833531856537, 1.007509708404541, -1.2294087409973145, 2.1927242279052734, -0.6449803113937378, -0.9337954521179199, 1.9301596879959106], [0.34057438373565674, -1.0680571794509888, -1.4066787958145142, 0.9921483397483826, 0.06492801010608673, 0.5540438294410706, -0.1227274164557457, -2.6226348876953125, -0.6653223633766174, 3.1865241527557373, 0.9210699200630188, -0.06308238953351974, 1.2699024677276611, -0.7941576838493347, 1.0553481578826904, -0.07365227490663528, -1.5284144878387451, 1.4181288480758667, 1.3450835943222046, 0.814472496509552, 0.29541653394699097, -0.023893307894468307, -1.2396141290664673, -0.14732077717781067, -0.26122066378593445, -0.8158160448074341, -0.5682453513145447, -0.5391733646392822, 1.3254607915878296, -0.8255376815795898, 2.1024441719055176, -0.7251946926116943, -1.442917823791504, -1.4177417755126953, -1.049961805343628, 1.1324453353881836, 0.3268444240093231, -1.40959632396698, 0.23005852103233337, -0.7644059062004089, 1.154296636581421, -1.4270234107971191, -1.648206353187561, 0.6129335165023804, -0.35100412368774414, -0.5470098853111267, 1.3070199489593506, -0.7933192849159241, 0.2632029950618744, 0.9099840521812439, -0.7418670058250427, 0.9059901237487793, 0.09708821028470993, 0.7097779512405396, -0.6881433725357056, -1.5524308681488037, -1.0806175470352173, -0.2855944335460663, -0.6281599998474121, 0.9769405722618103, 0.35388144850730896, -1.5773569345474243, 1.4899674654006958, 0.06281019002199173], [-1.5680954456329346, -1.2261666059494019, 0.6501659750938416, 0.3790178894996643, 0.3574136197566986, 0.5076532959938049, 0.16028328239917755, 1.517425537109375, -0.0686485543847084, -0.023148665204644203, 0.7060842514038086, 0.23504917323589325, 0.08268625289201736, 1.094160556793213, 0.07485257834196091, 2.8625612258911133, 0.17661653459072113, -1.0809283256530762, 0.37353330850601196, 1.8475638628005981, 0.47254762053489685, 0.5782954096794128, 0.7702692747116089, 1.1866363286972046, -0.9839362502098083, 0.24227258563041687, -0.25748637318611145, -0.09266330301761627, 0.039270609617233276, -0.43913090229034424, 0.07583299279212952, 2.3076658248901367, 0.30992212891578674, -0.417436420917511, -0.3080368936061859, 0.25313448905944824, 1.4643727540969849, 0.7240367531776428, 1.075669527053833, -0.6105254292488098, 0.5629537105560303, 0.9672962427139282, 0.4117412865161896, 1.4687000513076782, 0.8597769737243652, 0.7952724695205688, 0.8458490967750549, 0.22027097642421722, -1.085115909576416, -1.4595922231674194, -0.1494416445493698, -2.1935617923736572, 0.7392070293426514, 0.3072322905063629, -0.016757048666477203, 0.41605105996131897, -2.543989419937134, -0.33676642179489136, 0.7422778606414795, 1.6098568439483643, -0.03378758952021599, 1.186204195022583, -0.2384095937013626, 0.4518202841281891], [0.17526555061340332, -0.9131906628608704, -1.1507596969604492, -0.9211735725402832, -1.113899827003479, -0.9730199575424194, -0.04590662568807602, -0.14775346219539642, -0.2894214391708374, -0.2660068869590759, -0.7576815485954285, -0.10269795358181, 0.7543103098869324, -0.030789947137236595, -0.7736948728561401, -1.246199607849121, 1.0027306079864502, 1.7858071327209473, 0.5671119689941406, 0.8664771318435669, -0.021496566012501717, -1.0470243692398071, 0.5916492938995361, 1.3104033470153809, -0.8690069317817688, -0.47449207305908203, 0.03030744381248951, 0.20635128021240234, -1.4117769002914429, 0.11185789853334427, 0.3996575176715851, 0.13788817822933197, 1.9301912784576416, -0.22143688797950745, -0.504613995552063, 0.40985313057899475, -0.7572275996208191, -0.7678825855255127, 2.2765450477600098, -0.1809074431657791, -0.821127712726593, 2.098068952560425, -0.11210059374570847, -1.8735378980636597, -0.4633651375770569, 3.7545905113220215, 2.160597085952759, -1.7740923166275024, -0.8536866903305054, 1.1812572479248047, -1.3624228239059448, 0.5334473848342896, 0.7041444778442383, 0.7369796633720398, 0.014166994951665401, 0.26527276635169983, -0.8193018436431885, -0.05272471532225609, -0.5682211518287659, -0.8971980214118958, 1.6928002834320068, 0.7226894497871399, 1.631662130355835, 1.6240724325180054], [0.09453582018613815, -2.0015783309936523, -1.4580647945404053, 1.0092008113861084, -0.04460703209042549, 0.16180244088172913, 0.9288837909698486, -1.5436947345733643, -0.3263958692550659, 0.04836186021566391, -0.48350808024406433, -0.27063482999801636, 1.0248243808746338, 1.12433922290802, -0.474765807390213, 0.7016538977622986, 1.1117533445358276, -1.546800971031189, -0.8759575486183167, 0.6025012731552124, 1.1327269077301025, 0.48835793137550354, 0.3604067265987396, 1.107466220855713, 0.4063476324081421, 2.197256565093994, -1.3242396116256714, 1.8719642162322998, -0.08756859600543976, 1.2459514141082764, 0.13744154572486877, 0.9788733720779419, 0.0957435742020607, 0.2357671558856964, -0.874112069606781, -0.019480040296912193, 1.2530076503753662, 0.7869395613670349, -0.44475626945495605, 0.3020293712615967, -2.626277208328247, -0.0659426897764206, -1.5158847570419312, 1.6775317192077637, 1.8277441263198853, 1.8439974784851074, 0.08650172501802444, 0.058672841638326645, -0.13942615687847137, -0.2038014680147171, 1.1438473463058472, 0.5782312750816345, -1.2836006879806519, -1.4371706247329712, 1.3428761959075928, -0.501899778842926, -0.8623862862586975, -0.8156372904777527, -0.6811889410018921, -0.14893853664398193, 0.5835290551185608, -0.4784509539604187, 0.004154330585151911, 1.0029528141021729], [0.9853805303573608, 2.3028481006622314, -0.22907902300357819, 0.3524128496646881, -0.37886038422584534, 0.0686851367354393, -0.255321204662323, 1.9027236700057983, -0.3329075276851654, -0.154422789812088, -0.28601589798927307, -0.23582467436790466, 0.1581113636493683, 0.6192513108253479, -0.1202441155910492, -1.658586859703064, 2.3899285793304443, 0.18905510008335114, -0.5654679536819458, -0.5235117077827454, -0.6350369453430176, -1.7099064588546753, -1.1478520631790161, 0.9610949158668518, 1.0536620616912842, -0.04760909453034401, 0.24104636907577515, 0.8737420439720154, -0.21597547829151154, 0.7514007687568665, 0.18923114240169525, -0.877907931804657, -0.12755505740642548, 1.354197382926941, 0.019319219514727592, 1.4661213159561157, 0.39280903339385986, 0.005286120809614658, -1.4581667184829712, -0.2510814070701599, 0.7546224594116211, -0.43398407101631165, 0.770952045917511, -0.5603587627410889, 0.6022900342941284, 1.096903920173645, -0.029735147953033447, 0.35312384366989136, -0.6811121106147766, -0.09277995675802231, 1.7045575380325317, -0.3708450496196747, -0.9082017540931702, -0.28347304463386536, 1.032886266708374, -0.41911762952804565, -1.4714337587356567, -0.08066203445196152, -0.6032735109329224, 0.013695310801267624, -0.41622576117515564, -0.5863186120986938, -0.03450166806578636, -1.16361665725708], [0.24499407410621643, -1.4416615962982178, 2.441138982772827, 0.6952856779098511, -0.665790855884552, -0.2313290536403656, -1.038956880569458, 0.40328487753868103, 0.5965243577957153, 1.2398093938827515, 1.2558318376541138, -0.19938179850578308, 0.09086696058511734, -0.20301030576229095, 0.2201012521982193, 1.0433019399642944, -0.005017926450818777, -0.5195494294166565, 0.14664378762245178, -0.43045341968536377, -1.0355654954910278, -0.5716727375984192, -0.9444006681442261, -0.054481979459524155, 0.02933034859597683, -1.8755855560302734, 0.12529303133487701, 0.7069602608680725, 0.32000914216041565, -1.5595892667770386, -0.7192103266716003, -1.1856577396392822, 0.6024808883666992, 1.9492360353469849, 0.5517402291297913, 2.598172426223755, 0.09906533360481262, 0.4329985976219177, -0.5717892646789551, -0.04445084184408188, -2.4204249382019043, 0.6334881782531738, 1.2905604839324951, -2.730767250061035, 0.35811513662338257, -0.040823452174663544, 0.028773417696356773, -0.6623984575271606, 0.26030778884887695, -0.35217803716659546, -0.5028626322746277, 0.16236691176891327, -0.9347649812698364, 0.3683534860610962, 2.474059581756592, -1.418160319328308, -0.4155764877796173, -1.5645827054977417, -1.386925220489502, 0.034352611750364304, 0.12651661038398743, 0.17549721896648407, 0.018348002806305885, -0.5916258096694946], [0.9093884229660034, 0.4126776158809662, 0.3332895040512085, -0.7555898427963257, -0.4847029447555542, -0.5464232563972473, 0.6812087297439575, 1.0226434469223022, 1.287951111793518, -1.7345706224441528, -1.2039257287979126, 0.40165889263153076, 0.028043588623404503, 1.305286169052124, 0.7991899847984314, -0.4390918016433716, 0.5417144894599915, -0.674598217010498, 0.32737085223197937, -0.7548435926437378, -0.9082781076431274, 1.2784596681594849, 1.3079532384872437, -1.8244763612747192, 2.880695343017578, 0.4161255359649658, 1.1947546005249023, 0.17613963782787323, -0.16710014641284943, 1.5558176040649414, 0.5126445889472961, 0.42019158601760864, 1.602968692779541, -0.7242704033851624, 0.1650054156780243, 0.626230776309967, -0.10971743613481522, -0.5193496942520142, 1.509589672088623, 1.4445652961730957, -1.5091502666473389, 0.3216334283351898, -0.207960307598114, 1.010543704032898, 0.8128827810287476, -0.5454908609390259, -0.6391041874885559, -0.5728989839553833, -0.2728680372238159, 0.8010709285736084, -0.17867949604988098, -0.9892365336418152, 0.7175029516220093, 1.053977370262146, 1.2046884298324585, -0.06242114305496216, 0.8066504597663879, -0.4946655035018921, 1.5189236402511597, -1.4016361236572266, 0.9742858409881592, -0.48795634508132935, -0.4202723801136017, -0.4620126485824585]], [[0.6117432713508606, 0.3307538330554962, 1.719099760055542, -0.7908381819725037, 0.8784857988357544, -0.47951579093933105, 1.0802454948425293, 0.2824532687664032, 1.547455072402954, -0.12674593925476074, -1.0483514070510864, 2.168606996536255, 0.940952479839325, 0.7831851243972778, -0.15499186515808105, -0.21848852932453156, -0.49276164174079895, 0.49013692140579224, -0.5407795906066895, -1.914260745048523, 0.9994070529937744, 2.1128246784210205, 0.6923408508300781, -1.4110584259033203, -0.6620097756385803, 0.2575300335884094, 1.322436809539795, -0.2366291731595993, -1.317083716392517, -0.010801097378134727, -1.4721423387527466, -0.3210291564464569, 0.5935604572296143, 0.352486789226532, -0.6924688220024109, -0.42743632197380066, -0.5856929421424866, -0.39576616883277893, -0.4578709304332733, 0.4715898334980011, -0.18623052537441254, 0.5544925332069397, 0.8734636306762695, 1.363191843032837, 2.025125503540039, -0.13022653758525848, 1.3310446739196777, -0.963241696357727, 1.4485067129135132, 1.0459377765655518, -0.31058469414711, -0.001548977568745613, -1.427720308303833, -0.07755791395902634, -1.2124320268630981, 1.469999074935913, 0.6034814715385437, -0.17625464498996735, 0.4468453526496887, -0.1095811054110527, -0.05384023115038872, 0.5744187831878662, -0.513418972492218, -1.3597376346588135], [1.2610797882080078, 0.1622021645307541, 1.0659250020980835, -0.6257295608520508, 0.9943594336509705, -1.929425597190857, -1.1747099161148071, -0.5001630187034607, -1.2368146181106567, -0.3299981653690338, 0.4016243517398834, 0.39823925495147705, -0.9488258361816406, 0.5657315254211426, 0.9188989400863647, 0.06100428104400635, -0.8706819415092468, -1.0866105556488037, 0.5615800619125366, -0.43749287724494934, -1.0703085660934448, -0.7618817687034607, -0.953783392906189, 1.8416588306427002, -0.7265748977661133, 0.3985813856124878, 0.5797430276870728, 0.1289759874343872, -0.6001328825950623, 0.19157500565052032, -1.1096150875091553, -0.5099155902862549, 0.0015042823506519198, 0.6982952356338501, 0.29394665360450745, -0.4076201617717743, -0.06950968503952026, 0.07254332304000854, -0.5040217638015747, 0.1327599734067917, -1.045316219329834, -0.5656304359436035, -0.15512505173683167, 1.123058557510376, -1.097690463066101, 0.8297274112701416, -0.30077478289604187, -0.3215172290802002, 0.7626104950904846, 0.09895892441272736, 1.0377404689788818, -0.9279342889785767, -0.8638995885848999, -0.3008611500263214, -0.12544666230678558, 1.042510747909546, 1.3948190212249756, 2.3855342864990234, 0.09519315510988235, -0.28648465871810913, -1.7325999736785889, -0.15880808234214783, -0.02989904396235943, 1.606360912322998], [0.6196789145469666, 0.893474280834198, 0.5350542068481445, -0.8954226970672607, -1.2786463499069214, 0.2305578887462616, -0.5466042757034302, -1.3636795282363892, -0.0836743712425232, 0.5219612121582031, 0.6526864171028137, -1.275860071182251, 1.5139663219451904, -0.35137584805488586, 1.5727086067199707, -0.5589144825935364, 0.2851024866104126, -0.5638794898986816, 0.5935974717140198, -1.2036594152450562, 0.004781309049576521, 0.971307635307312, -0.09043430536985397, 0.8708606362342834, 0.1941695362329483, 1.2696675062179565, 1.5186153650283813, -0.8863188028335571, 0.0422031544148922, -1.1952134370803833, -0.13879798352718353, -0.3746224641799927, -0.2961420714855194, -1.0876222848892212, 2.1036970615386963, 0.16872933506965637, -1.2715858221054077, 0.17257677018642426, -0.0600522942841053, 0.7457751035690308, 2.3427605628967285, 1.895795464515686, -0.8921986222267151, 0.42326319217681885, 0.10647274553775787, 0.5399969220161438, -0.7632986307144165, -1.1476078033447266, 0.8240695595741272, 1.1224756240844727, -0.13144780695438385, 1.2686514854431152, -1.4415686130523682, -0.29990166425704956, 0.70405513048172, 0.24875588715076447, -0.37554094195365906, -0.16674432158470154, 0.8455114960670471, -2.0363996028900146, 1.1411443948745728, -0.9132367968559265, 0.17793340981006622, 0.6634895205497742], [0.29133671522140503, 0.34819456934928894, -0.8149849772453308, -1.340802550315857, 1.7122318744659424, 0.17274466156959534, 0.787248432636261, 0.8827417492866516, 2.1594862937927246, -1.2643367052078247, 2.4786338806152344, 0.4989679157733917, -1.4617087841033936, -1.1927217245101929, -0.7678903937339783, -0.9833356738090515, 0.13090503215789795, 0.12597812712192535, -0.9208294749259949, 0.128226637840271, 2.1825098991394043, -0.9891469478607178, 0.41121581196784973, -0.8094167113304138, 0.7086142897605896, 0.0007269051275216043, 1.561058521270752, -1.0608186721801758, -1.2098277807235718, -0.23781119287014008, 0.07851480692625046, 0.6444214582443237, 0.4054326117038727, -1.015671730041504, -0.9476436376571655, -0.9416036605834961, 0.8869562149047852, -0.8112519383430481, -0.052100662142038345, 1.0570225715637207, 0.4010462462902069, -0.33372822403907776, 0.6219961047172546, 0.7951924204826355, -0.37440183758735657, -0.4406386911869049, -1.7185693979263306, 0.7341101765632629, 1.0722265243530273, 0.44185805320739746, 0.7343413233757019, 0.2864750027656555, -0.7877167463302612, -0.01321711391210556, 2.097642183303833, 0.7285893559455872, -0.6734583377838135, -0.09775689244270325, 0.21009992063045502, 0.5349825620651245, 1.5255190134048462, -0.6564925312995911, 0.5944488644599915, -0.10979504883289337], [-0.2563685476779938, -1.1611230373382568, -0.5334506630897522, -0.8268241286277771, 0.07706084847450256, 0.9245995283126831, -1.79026198387146, 0.726878821849823, 0.947252631187439, 0.4765002429485321, 0.9189902544021606, 1.0294126272201538, -0.42775759100914, 0.06108104810118675, -0.6519219875335693, 1.3057018518447876, 0.47094979882240295, -0.7177333831787109, -1.1499528884887695, 0.5728737115859985, 1.627364158630371, 0.5052794814109802, -0.22764882445335388, 0.5493429899215698, 1.7045947313308716, 0.6696248054504395, 0.8022226095199585, -0.2839326858520508, -0.1284743994474411, 0.3435373604297638, -0.6634503602981567, 0.4942722022533417, -0.7834247350692749, 2.309488296508789, 0.9379700422286987, 0.09456387162208557, 0.8286904096603394, 0.42229869961738586, 1.264223337173462, -0.49760064482688904, -0.21020589768886566, -0.7678739428520203, 0.7171269059181213, -0.4624868333339691, 0.10254267603158951, 0.7649878263473511, -0.8542622923851013, 2.581150531768799, 0.20574074983596802, -1.6334881782531738, 1.3368782997131348, -2.0698134899139404, -0.8324936032295227, -1.4497652053833008, 1.547279953956604, -0.7721930146217346, -0.7415318489074707, 0.47476401925086975, 0.14129725098609924, -0.498922199010849, -1.396567463874817, -1.362375259399414, 2.127784490585327, 0.7028341889381409], [1.574469804763794, -0.5705477595329285, 0.7261847257614136, 0.5441359877586365, 1.5105489492416382, 0.4052809178829193, 0.18724998831748962, 0.45098644495010376, -1.3905514478683472, -0.05148174986243248, -0.27509579062461853, 2.0225038528442383, -0.40389740467071533, 1.2886384725570679, 1.7476253509521484, -0.32737040519714355, -0.9801255464553833, -0.9850638508796692, 0.48044857382774353, -0.6371172070503235, -0.01014840416610241, 0.3733310103416443, 0.08426021784543991, -1.558786153793335, -1.8515253067016602, -0.200914666056633, -0.09325302392244339, -0.6511587500572205, 1.2411651611328125, 1.0139038562774658, 0.03858187794685364, -0.2254752218723297, 0.705336332321167, 0.081965371966362, -1.092596173286438, 1.1023430824279785, -0.43263038992881775, -1.1282604932785034, -0.7154102921485901, -1.4215904474258423, 0.37978997826576233, 0.478644996881485, -1.8022581338882446, 1.7526981830596924, 0.5375781059265137, 0.5240506529808044, -2.7090396881103516, 2.123511552810669, 2.4499850273132324, 0.8212078809738159, 0.02878270298242569, -0.8028417229652405, -0.8852238059043884, -1.5470821857452393, -0.8367324471473694, 0.7446463108062744, 0.6111398339271545, 0.6916578412055969, 2.199479818344116, 0.3045819401741028, -0.9473174810409546, 1.06771981716156, 0.6586836576461792, 0.41322755813598633], [-0.25975438952445984, 1.2012661695480347, 0.3859541118144989, 1.513479232788086, 0.41686591506004333, -1.05305814743042, -0.3758375346660614, 1.3912492990493774, -0.42736804485321045, -0.44542595744132996, 1.340572476387024, -0.008340488187968731, -1.1319077014923096, -1.161372184753418, 0.03538142517209053, 0.5253288745880127, 0.41251465678215027, -0.2879208028316498, -1.7062922716140747, 1.1490094661712646, 0.2512683868408203, -0.18504273891448975, 0.41231584548950195, -1.453011393547058, -1.148270845413208, -0.06146597862243652, -0.6165775656700134, 0.6655429005622864, -0.23462098836898804, -0.10184692591428757, 0.31414246559143066, 0.22472336888313293, 0.057330742478370667, -0.2911575734615326, -0.28559064865112305, 0.4921071231365204, -0.6334257125854492, -1.132347583770752, -0.8758174777030945, 0.9589250087738037, -0.4263562560081482, 0.16420283913612366, 0.14000436663627625, 1.794144868850708, -0.7939204573631287, 1.1331992149353027, -0.8725574016571045, 0.6220085620880127, 1.6719824075698853, 1.096543788909912, 0.18043839931488037, 1.7548339366912842, -1.0919972658157349, -0.023213285952806473, -0.42239251732826233, 1.3486319780349731, 0.029535464942455292, 0.0012617438333109021, 0.7449546456336975, -1.0843534469604492, 2.0064682960510254, -0.886530876159668, -0.15870226919651031, -0.04305250570178032], [0.6507193446159363, 0.9048263430595398, -0.2806227505207062, -0.7466864585876465, 1.1647893190383911, 1.3172268867492676, 0.08914587646722794, -2.698672294616699, 1.2572643756866455, 1.7551847696304321, -1.5809667110443115, -0.6091871857643127, 0.9597101211547852, 0.7224075198173523, -0.04991799220442772, -1.8484777212142944, 0.33211788535118103, 0.6157134771347046, -0.9150558114051819, 0.037223391234874725, -0.11154726147651672, 0.2993164360523224, -0.7086700201034546, 1.9281985759735107, -1.37445068359375, -0.2663891613483429, 0.8613665103912354, 0.37095412611961365, -0.5357948541641235, -1.0719401836395264, -0.8806555271148682, 0.5635249614715576, 0.13599660992622375, -0.9358574151992798, -2.054690361022949, 0.47674524784088135, -0.050274722278118134, 0.8654171228408813, -2.393645763397217, 0.6167794466018677, -0.7046111226081848, 0.38958126306533813, -0.6504655480384827, 0.06498895585536957, -1.7493932247161865, 0.9211990237236023, 1.0037485361099243, -0.32412609457969666, -1.958371877670288, 1.33308744430542, 1.1328750848770142, 0.2587408125400543, 0.9884172081947327, -0.9687871336936951, 1.9005359411239624, 0.00985549483448267, 2.86179256439209, -0.37352126836776733, 0.5938159227371216, -0.2692272663116455, 0.2084045559167862, -0.8016931414604187, -0.09959963709115982, 0.5138639211654663], [0.40122073888778687, 0.9293825626373291, -1.1313037872314453, 1.1331714391708374, -2.111051559448242, -0.3604864776134491, 0.19846731424331665, -1.2077959775924683, -2.9180386066436768, -0.1568303257226944, -0.4671393930912018, 0.6604037880897522, 0.6489934921264648, -1.5652586221694946, 0.3425333797931671, -0.29151812195777893, -0.6398760676383972, 0.9977406859397888, -0.5202412605285645, 0.8652810454368591, -0.1749047040939331, -1.652940034866333, -0.09210321307182312, -1.1974760293960571, -0.8740143179893494, -1.9622143507003784, -0.5478892922401428, 1.0735944509506226, 0.5700066685676575, 0.18349769711494446, -0.5498981475830078, -1.535059928894043, -0.24759095907211304, 0.2743837237358093, -0.42601075768470764, -0.9602908492088318, 0.8861318826675415, 0.9400697946548462, 0.8325326442718506, 0.45431479811668396, 0.22062894701957703, 0.7449164390563965, -2.6603989601135254, 0.22416619956493378, 0.8294802308082581, 0.05832940712571144, -0.9769405722618103, -0.1687294989824295, 2.977581024169922, 1.2764455080032349, -0.9302740693092346, 0.8511533141136169, 1.791015386581421, -0.8693352937698364, 0.2244851142168045, -1.338088870048523, -0.1390024721622467, 0.37080317735671997, 0.9631152153015137, -1.2267258167266846, -0.6673558354377747, 0.5948562026023865, 1.4404501914978027, 0.8025141954421997], [1.3209643363952637, -0.19575177133083344, 2.1661720275878906, -1.0548686981201172, -0.11883620917797089, 0.36616528034210205, -1.949117660522461, -1.9768917560577393, 1.7638139724731445, 0.3219478130340576, 0.9300961494445801, -1.2549619674682617, -1.586333155632019, -1.8212629556655884, -0.39690542221069336, -1.098192811012268, 0.21320921182632446, 1.2173513174057007, 0.25473475456237793, 0.48812001943588257, -2.0160281658172607, -1.4605426788330078, 2.207674503326416, 0.13514915108680725, -0.5561659336090088, -1.7760878801345825, 2.2848777770996094, -1.4444833993911743, -0.6913607716560364, -0.47760164737701416, -1.081875205039978, 0.4240390658378601, 0.8840951323509216, 1.0509251356124878, 0.03792191669344902, 0.09059120714664459, -2.1326584815979004, -1.2035515308380127, -1.1553223133087158, 0.35743486881256104, -1.1416237354278564, 0.6834484338760376, -0.414051353931427, -1.5006132125854492, 0.12748752534389496, 1.2491804361343384, 0.79909348487854, -1.3214118480682373, 1.1702206134796143, -0.5781773328781128, 0.6296591758728027, -0.2938775420188904, 0.6393259763717651, -1.3581620454788208, -0.1609801948070526, -0.40059757232666016, 1.2707133293151855, -0.9297778606414795, -0.3191225528717041, -1.2941062450408936, -0.004668351728469133, 0.40292730927467346, -1.7929682731628418, -1.1170262098312378], [0.7086396217346191, 0.5895422101020813, 0.9729563593864441, 1.4866887331008911, 0.06667663156986237, -1.036597490310669, 1.2006776332855225, 0.9113695621490479, -1.026606798171997, -1.0161269903182983, 0.4326428174972534, 0.11256548017263412, 1.270849347114563, 0.3767235577106476, 0.179023876786232, -0.7001223564147949, 1.9900202751159668, -0.0452120304107666, -0.19891691207885742, 0.18248294293880463, -0.28082019090652466, -0.31760942935943604, 0.01991775445640087, 0.12653526663780212, -0.32907578349113464, -0.14953064918518066, -0.9450263381004333, -0.28994014859199524, 2.397185802459717, 0.7111168503761292, 0.08698765933513641, 0.8357647657394409, 0.4684104919433594, -1.030336856842041, 0.5200878977775574, -1.1662989854812622, 2.1611204147338867, -0.0905655026435852, 0.31704989075660706, -0.682661771774292, -0.8438066244125366, 0.3298890292644501, 0.5399900674819946, -1.1233761310577393, 0.07468333095312119, -2.070181369781494, 0.21538420021533966, -0.65260249376297, -0.048578765243291855, 1.3060352802276611, 1.276898741722107, 1.0129481554031372, 0.12491258978843689, -0.3741259276866913, 1.2206134796142578, 0.3154789209365845, 1.2584428787231445, 1.2133889198303223, 0.48620882630348206, -0.9453436732292175, -1.041632890701294, 0.8175331354141235, -1.192070484161377, 0.6488847732543945], [-1.7131770849227905, -0.089823417365551, -1.2736390829086304, 0.25750732421875, -1.0384107828140259, 0.008930769748985767, -0.7440053820610046, -0.8403428792953491, -0.386893630027771, 0.10656683146953583, -0.8630138039588928, -0.6869087219238281, -0.007120821159332991, -0.9593808054924011, -1.0286685228347778, 0.38392820954322815, -0.45715856552124023, 0.46200111508369446, -0.42548444867134094, 0.9790414571762085, 1.5015062093734741, 0.5419349074363708, 0.0010868889512494206, -0.5904126763343811, 1.0364789962768555, -0.41312119364738464, 0.9307724833488464, -0.21967901289463043, 0.7096748352050781, -0.6499179601669312, -2.0347869396209717, 0.570990800857544, -0.472374826669693, 2.3030664920806885, -1.013085126876831, 0.7817960381507874, 1.6489850282669067, 0.46113118529319763, -0.2904636561870575, 1.8608592748641968, -0.26522064208984375, 1.3431025743484497, -3.09420108795166, 0.23057998716831207, -0.19447749853134155, -0.6668264865875244, -1.206852912902832, -1.0793139934539795, 1.5914313793182373, -0.04552832990884781, 0.08071563392877579, 0.7935624718666077, 0.6194254755973816, -1.4427953958511353, 1.286824107170105, -0.37935733795166016, 1.0995044708251953, -0.830790102481842, -0.8112942576408386, -0.06938832253217697, 1.3556413650512695, -0.29022103548049927, 0.7605552077293396, 0.3477635085582733], [-1.9339443445205688, -0.482179194688797, -1.9973177909851074, -0.2115863859653473, 0.004010844975709915, 0.051128800958395004, -0.7974879741668701, 2.467653512954712, 0.9780679941177368, 0.29941490292549133, -0.8047670722007751, -1.7800508737564087, -0.28828951716423035, -0.00903377402573824, 0.09980174154043198, -0.8228432536125183, -0.8071406483650208, -0.5349491834640503, 0.24340878427028656, -0.3384351432323456, 0.2056809663772583, -0.18007780611515045, -1.993038296699524, 0.958365261554718, -1.6903746128082275, -1.1313680410385132, 0.059515755623579025, -1.655645489692688, -0.5334650278091431, -1.0648343563079834, 0.7761645913124084, 0.7115360498428345, 1.5836633443832397, 0.15964558720588684, 0.20240439474582672, -0.4814683496952057, -1.3692926168441772, 0.8965765237808228, 0.3216054439544678, -0.32773253321647644, 0.27597111463546753, -0.6252961754798889, 1.1548993587493896, -1.6821794509887695, -1.1816375255584717, -1.046510934829712, -0.04964692145586014, -0.8591837286949158, 1.7346075773239136, -0.5405699014663696, -0.05212768167257309, -1.1737600564956665, -0.47222498059272766, -0.7293120622634888, -0.10817720741033554, -1.8881055116653442, -1.4024369716644287, 1.7109113931655884, -0.9056031703948975, -0.07238274812698364, 1.407344937324524, -0.32481372356414795, 0.855211079120636, 0.48898524045944214], [-0.3151620328426361, 0.5650559067726135, 1.5769292116165161, 0.4794299900531769, 0.4466516077518463, -0.5836386680603027, 0.42916926741600037, -0.020769864320755005, -1.067381501197815, -1.0822129249572754, -0.9937295913696289, 0.4257412850856781, -0.48852285742759705, -0.1529262214899063, -1.4030331373214722, 1.2461498975753784, -1.3626285791397095, 1.3974553346633911, 0.30319100618362427, -0.7391207814216614, 1.010229468345642, 0.6961972117424011, -0.28525489568710327, 0.4908912479877472, 1.8601562976837158, 0.028065018355846405, 0.3883231282234192, 0.16267095506191254, -0.38696110248565674, -0.09311620146036148, 0.9479560852050781, 0.1781514286994934, 0.1759747862815857, 0.9967004656791687, -0.34921014308929443, 1.2428789138793945, 0.7178913354873657, 0.396836519241333, 0.23967309296131134, -0.44156917929649353, -1.165794014930725, -0.3304893672466278, 0.2334575057029724, -0.7049907445907593, -0.9596838355064392, 0.4497007429599762, -0.5002169013023376, 0.7135840058326721, 0.4488650858402252, -0.5630092024803162, -0.25209230184555054, 0.7935229539871216, -0.8829476833343506, 0.6560383439064026, 0.11512833088636398, 0.23785026371479034, 1.0397570133209229, -0.46139073371887207, -2.1356301307678223, -0.09151952713727951, 0.4702100455760956, -0.8110668063163757, 2.96852970123291, 0.07379233837127686], [-0.29451385140419006, 0.1400361955165863, 2.1323745250701904, -0.3037891089916229, 1.4929429292678833, 0.7187389135360718, -1.822044014930725, -2.7635281085968018, 3.1088974475860596, -0.6714189648628235, 1.7742440700531006, 0.8401896953582764, 2.3783602714538574, -0.4722394645214081, -1.5549628734588623, -1.321537733078003, -1.1384730339050293, -0.40786921977996826, 1.3025394678115845, -2.2526652812957764, -1.134633183479309, 1.1615124940872192, -0.11567389965057373, 0.46498024463653564, -0.6986736059188843, -2.5476677417755127, 0.20710226893424988, -0.19290708005428314, 0.10545984655618668, 0.13269317150115967, 0.2685379087924957, 0.01942451111972332, 0.3539404571056366, -1.0807663202285767, 1.8976088762283325, 1.107832670211792, -0.4740369915962219, -1.1043702363967896, 1.1123465299606323, 1.0686720609664917, -0.6592000722885132, -0.9556788206100464, 0.48190316557884216, -2.1535120010375977, -0.1491282731294632, -0.49590781331062317, 1.730772614479065, -0.9625909328460693, 0.34160149097442627, -0.8077349662780762, -2.8047616481781006, 1.8508226871490479, -1.348455786705017, -1.0941418409347534, 1.039678931236267, 0.37689143419265747, -0.7922956943511963, -0.6039923429489136, -0.16897624731063843, -1.4182299375534058, -0.3471343517303467, -0.27951306104660034, 0.8000428080558777, 0.9767970442771912], [1.664932370185852, 1.0946400165557861, 0.7940214276313782, 0.21046574413776398, -0.5238152146339417, -0.770175576210022, 0.1818896234035492, 0.06491602212190628, -0.33107849955558777, -1.1957008838653564, 0.6030930876731873, -0.6318342685699463, 0.9338067770004272, -0.253030925989151, 1.7419860363006592, 0.20007847249507904, 0.5107613205909729, -0.9752915501594543, 0.46706080436706543, 0.5789914131164551, -0.013652723282575607, -0.5696702003479004, -0.7244613766670227, 1.7043848037719727, -1.513835072517395, -0.31158319115638733, -0.7230713963508606, 1.1941932439804077, 0.1068049967288971, -0.6810458302497864, -1.671629786491394, 0.7737864851951599, 1.2480006217956543, -0.30551043152809143, 0.48210588097572327, 0.40569907426834106, 0.7244998812675476, 0.1796293407678604, -1.5450764894485474, 0.09294870495796204, 0.07107627391815186, -0.18804262578487396, 0.6643715500831604, 1.1021080017089844, 1.0943408012390137, 1.0145988464355469, 0.20188483595848083, 0.6197437047958374, -0.48985204100608826, -0.5971758961677551, 0.793407678604126, 0.9824494123458862, -0.12041948735713959, -0.9171987175941467, 0.8243722915649414, -0.47911933064460754, 0.17388483881950378, -1.5686978101730347, 0.2119312584400177, -0.5990415811538696, -0.5564956068992615, 1.0077213048934937, 0.06068456918001175, 0.025167331099510193]], [[0.5510475039482117, 0.45110490918159485, -0.8377856016159058, -0.5412324666976929, -0.252497136592865, -0.0010483141522854567, 0.07363682240247726, -2.03886079788208, 0.6347575187683105, 1.2964714765548706, -1.9854973554611206, -0.6681477427482605, -0.4690188467502594, -0.781452476978302, 0.0035302313044667244, -1.7321592569351196, 0.04024379327893257, -0.00016424716159235686, -1.4072493314743042, -2.115900754928589, 0.30852463841438293, -0.9470226764678955, 1.3366076946258545, 0.08500143885612488, -0.7593587636947632, 1.2899668216705322, 0.02153971791267395, -0.5880048871040344, -0.5340512990951538, -0.812881350517273, -0.2646431028842926, -0.7809235453605652, 0.13112977147102356, 0.631677508354187, -0.1647246927022934, 0.7774410843849182, -0.9439135789871216, -0.8530406951904297, -0.5879764556884766, -0.37484848499298096, 1.2267141342163086, -0.4150840640068054, -0.8463662266731262, 0.5445119142532349, -2.636843204498291, 0.7540192008018494, -1.891639232635498, -0.643644392490387, -0.572792112827301, -1.0421546697616577, -1.316263198852539, 1.2171107530593872, -2.7774336338043213, -0.5773075222969055, -0.8058428168296814, 0.22650004923343658, -2.2602500915527344, 0.3462602198123932, 1.8800811767578125, 0.6524237394332886, 0.6513301134109497, 0.08925744146108627, 0.24016845226287842, 0.24835263192653656], [0.30260559916496277, 0.34909769892692566, -0.5110004544258118, -1.8791571855545044, 1.4301621913909912, 1.2738665342330933, -0.19122380018234253, 1.6070188283920288, -2.104172468185425, -0.02902982197701931, 1.2189048528671265, -0.475911408662796, 1.009049654006958, -0.5831737518310547, 2.1954734325408936, 1.2904866933822632, 0.5581377744674683, -1.0374023914337158, -0.328338086605072, -0.7799020409584045, -0.7037825584411621, 0.0005521723651327193, 1.1118803024291992, -1.5933314561843872, 1.1048924922943115, -1.1856869459152222, -2.3288989067077637, 0.8117262721061707, 0.5990626215934753, -0.4146921932697296, -0.260614275932312, -0.6904717683792114, -0.14081920683383942, -0.8010792136192322, -0.4849558174610138, -0.642362117767334, 1.7079354524612427, 0.2852577567100525, -1.447368860244751, -0.17525383830070496, -1.1815091371536255, 0.22726836800575256, 0.295684278011322, 0.6938281059265137, -1.4073041677474976, 1.1122862100601196, 0.7581015825271606, 0.005540505982935429, 1.8129557371139526, -0.8256434202194214, 1.0499327182769775, 0.7917920351028442, -0.26509445905685425, -0.027607988566160202, -0.42344558238983154, -0.1753142923116684, 0.3506934642791748, 0.28156203031539917, 1.4291377067565918, -0.554897129535675, 0.6172391176223755, -0.08227526396512985, 0.9045560956001282, -0.7245833277702332], [-0.014005755074322224, 0.5506020784378052, -1.205862283706665, 0.8830478191375732, 0.2106158584356308, -0.9523411989212036, -0.24264249205589294, -1.4264289140701294, 0.5919209718704224, -0.035958368331193924, -1.3632915019989014, 1.9183968305587769, 0.8139184713363647, 0.6912352442741394, -1.1060786247253418, -1.3279885053634644, 0.8930712342262268, 0.45117470622062683, 0.3090292811393738, -1.5054609775543213, 0.38911503553390503, 0.678068220615387, 1.1125521659851074, -1.8905545473098755, -0.8393733501434326, 0.5803723931312561, -1.2545429468154907, -1.0913734436035156, 1.1080429553985596, -0.39175036549568176, -0.07283896952867508, -0.8099192976951599, 0.33949998021125793, -1.1176103353500366, -1.4063470363616943, 0.28770768642425537, 0.049175046384334564, 0.12445104867219925, 0.6032111048698425, -0.2768835127353668, 0.09747668355703354, 2.8831615447998047, 0.5362674593925476, -0.3953608572483063, -0.3291723132133484, 0.05519359931349754, 0.38147398829460144, -0.38365691900253296, 0.8168781995773315, 0.19143112003803253, -0.09978233277797699, 0.7715660333633423, -1.207231879234314, -0.9477736949920654, 0.5808735489845276, -1.8308717012405396, -0.7312027812004089, -0.6638030409812927, 0.005306030623614788, -1.8534966707229614, 2.191554546356201, -0.9073959589004517, 0.045727912336587906, 1.3797714710235596], [1.3902515172958374, 0.2877681851387024, 0.8251571655273438, -0.8646895885467529, -0.9645817279815674, -0.9521104097366333, -0.584136426448822, -0.06856338679790497, 1.1165497303009033, 0.9427222609519958, 0.485003262758255, 1.366707444190979, 0.194069504737854, -0.0018582851625978947, 0.1368139535188675, 0.7810993790626526, -0.23472657799720764, 0.1663742959499359, 0.09246938675642014, -0.828126847743988, -0.4625411927700043, 0.228130504488945, 0.023995762690901756, -0.8044687509536743, 0.8513607382774353, 1.8220280408859253, -0.02640298195183277, -0.6419013142585754, -0.6837194561958313, 0.4836875796318054, 1.5931085348129272, -0.1575937569141388, -0.10329674184322357, -0.39087799191474915, -0.01866118423640728, 0.5085265040397644, -0.36690378189086914, 0.3342169225215912, -1.0426881313323975, -1.6472065448760986, 0.45222100615501404, -0.29437538981437683, 0.14926353096961975, 0.6788806915283203, 0.36435237526893616, -2.1169023513793945, 2.054488182067871, -0.14674368500709534, 0.21495398879051208, -0.7817699909210205, -0.03650953993201256, -0.37467315793037415, 0.8780931234359741, -1.2165995836257935, -0.3227832615375519, 0.47851625084877014, 0.2769787907600403, -1.022487998008728, -0.10143566131591797, -1.5273079872131348, -0.5196540951728821, 0.49105024337768555, 0.48036089539527893, -0.6432353258132935], [1.0643186569213867, 0.1573655605316162, 0.3989911377429962, -0.3989602327346802, -0.3958275020122528, -0.0925508365035057, -0.13002899289131165, -0.7706928253173828, 0.17705638706684113, -1.1993063688278198, -1.79475736618042, 0.7261229753494263, -1.480637550354004, -0.8267199993133545, -1.700175166130066, 1.6424461603164673, -1.1598260402679443, -0.39095863699913025, 0.1107177585363388, -0.09072522073984146, -0.7566426396369934, 0.10090538114309311, -0.24051609635353088, -1.4592689275741577, 0.17699608206748962, -0.01208486221730709, 0.7569180130958557, 0.15776799619197845, -1.1716476678848267, 0.09810660779476166, -0.6337353587150574, 1.4744395017623901, 2.1917905807495117, -1.328479290008545, -0.6298107504844666, -1.6234604120254517, 0.8882924318313599, 0.4652755856513977, -1.5073851346969604, 0.09929847717285156, 0.13940604031085968, -1.9254206418991089, -0.46409040689468384, -1.2537846565246582, -1.074873447418213, -0.3547980487346649, -1.1564325094223022, -1.1073707342147827, 0.4640437960624695, 0.12283675372600555, -0.6612945795059204, 0.8419765830039978, -1.845766544342041, 0.9037477970123291, -0.6594877243041992, 0.039953432977199554, -0.43466809391975403, -0.5299076437950134, 1.7909115552902222, 1.0271244049072266, -2.0898196697235107, 0.41927847266197205, -0.018170377239584923, -0.1045914813876152], [-1.164932131767273, -0.7037194967269897, -1.587228775024414, 0.6902994513511658, -1.5271774530410767, 0.3830691874027252, 0.3414325416088104, -1.4421495199203491, 0.3521881699562073, 0.9412959218025208, -0.31095829606056213, -1.3687810897827148, -0.45291605591773987, 0.07037026435136795, -1.6491390466690063, -1.337557077407837, 0.648242712020874, -0.7836018800735474, -0.44464269280433655, -1.471659779548645, -0.9057486653327942, -0.9958221316337585, 0.1272449642419815, -0.40837687253952026, -1.0024700164794922, 0.4849955141544342, 0.3007809817790985, 2.7728934288024902, -0.09259457886219025, -0.48056861758232117, -1.4112541675567627, 1.3434946537017822, -1.4260202646255493, -0.33875882625579834, 0.5422768592834473, -0.46669861674308777, 0.733253002166748, 0.009383419528603554, 1.1042320728302002, 0.05153731256723404, 1.3069331645965576, 0.05970528721809387, 0.25717002153396606, 1.029707670211792, 1.2819838523864746, -0.6573824882507324, -1.5797295570373535, 0.4759277105331421, 1.3199628591537476, 0.4970085322856903, -0.03761213272809982, -1.409738540649414, 0.23624002933502197, -0.3585197627544403, 0.10868428647518158, 1.9872941970825195, -0.5601415634155273, 0.4162561595439911, -0.38593316078186035, 0.1400420367717743, -0.9092892408370972, -0.1304996907711029, 0.01321831438690424, 1.8224527835845947], [-0.644195556640625, -1.0056735277175903, -0.24654924869537354, -0.917263925075531, -0.6691651940345764, -0.7546720504760742, -0.01894083432853222, -0.7954978942871094, 0.08745197206735611, 0.41709983348846436, -0.12330224364995956, -1.3289775848388672, 1.5585880279541016, -0.14756856858730316, 0.37792280316352844, -0.0656026154756546, 0.961126983165741, -0.6831126809120178, -0.2244388312101364, -1.4061411619186401, -0.006935416720807552, 0.8358990550041199, -0.34292155504226685, 0.029259420931339264, 0.8082229495048523, 0.7499920725822449, 0.04334905743598938, -0.8742557168006897, -0.6649061441421509, 0.37855759263038635, 0.09986475855112076, 0.6769981384277344, 0.541644811630249, 0.31280314922332764, -1.080645203590393, -1.6872127056121826, 0.4831497073173523, -1.5054305791854858, 0.16438645124435425, -0.21051427721977234, 1.1081602573394775, -1.6361697912216187, -0.5646700263023376, 0.0018348683370277286, -0.6533657908439636, -0.6310514807701111, 0.24409230053424835, 0.08163636177778244, -0.7913898229598999, 0.6240264177322388, 0.5542711615562439, 0.1548151820898056, -0.9712766408920288, 0.8554818630218506, -1.0802942514419556, -0.1429230272769928, 0.6926798224449158, -0.40655508637428284, 0.5760378837585449, 1.463870882987976, 1.2525599002838135, -0.5070398449897766, -1.3451858758926392, 1.19225013256073], [0.15919016301631927, 0.18688024580478668, -1.0014837980270386, 1.726847529411316, 0.38677334785461426, 0.0354803241789341, 0.907650887966156, -0.6733133792877197, -0.4968298375606537, -1.0524959564208984, 0.9749950766563416, 1.222572922706604, 0.35948464274406433, -0.18336640298366547, -0.27849280834198, -0.15066400170326233, -1.0818127393722534, -0.0705007016658783, -0.1837126910686493, 0.021552419289946556, 0.7063227295875549, -1.5606335401535034, 0.9036967754364014, -2.3585498332977295, -0.07385601103305817, 0.13400501012802124, 0.2764730155467987, 0.6165434122085571, 0.5816645622253418, -0.1721956878900528, -0.1647203266620636, 0.7245743870735168, -0.9145128726959229, 1.7208539247512817, -0.25184014439582825, -0.3160243332386017, 1.144935131072998, 0.5599855780601501, -0.4319770038127899, -1.021064043045044, 0.0007565958076156676, -1.2204073667526245, 0.6891810894012451, -1.2428014278411865, 1.532334804534912, 0.40342918038368225, 0.3572349548339844, -0.7530184388160706, -0.2569214701652527, 2.183046579360962, -0.7019652128219604, -0.7051798701286316, -0.839887797832489, 0.18316343426704407, 1.1796687841415405, -0.061110299080610275, 1.035402536392212, 2.691915988922119, 0.08871069550514221, -1.2244155406951904, -0.03995512053370476, 0.4030817151069641, -0.24466168880462646, 0.594307541847229], [-0.2182890623807907, 0.6241633296012878, -1.34402596950531, -0.3454959988594055, 0.17152492702007294, 0.39375218749046326, 0.4111478626728058, 0.04674327000975609, 0.19074752926826477, -0.7779505848884583, 0.33213844895362854, 0.42779043316841125, 0.34812694787979126, -0.5515667796134949, -1.7312177419662476, 0.19756774604320526, 0.4045398235321045, 1.0847218036651611, -0.38402536511421204, -0.3499128222465515, -0.4235685467720032, 0.5410435795783997, -1.7539551258087158, -0.7823672890663147, -1.7989745140075684, 1.138404369354248, 0.40235936641693115, -1.2751163244247437, -1.4063719511032104, -0.6298544406890869, -1.902387261390686, -0.9246838688850403, 0.6569748520851135, 0.0600469633936882, 0.627687394618988, -0.23066525161266327, 0.10513556003570557, -1.4544121026992798, 1.5794473886489868, -0.6269994378089905, -1.1410058736801147, -1.729764699935913, -1.6328346729278564, 1.4146215915679932, 0.6434935331344604, 0.5232924818992615, 0.21307605504989624, -0.24444352090358734, 0.580935537815094, -0.3380068838596344, -0.1673399955034256, -0.532466471195221, 0.6211300492286682, -0.461355060338974, -1.1646647453308105, 1.512121319770813, 0.6430439949035645, 0.1150447204709053, -0.5267953872680664, 0.9950847625732422, 1.3892589807510376, -1.4811536073684692, 1.1646974086761475, 2.130697011947632], [-0.20089088380336761, -0.6437857747077942, -0.25129154324531555, 0.31864920258522034, 1.2365000247955322, -1.8039367198944092, -0.608400285243988, -0.06383766233921051, -0.26052385568618774, -0.5794777870178223, 0.5601658821105957, -0.5350218415260315, -0.48181647062301636, -1.1867820024490356, 1.4401073455810547, -1.549237608909607, 0.8572079539299011, -0.1970854550600052, 0.059172630310058594, 0.34533706307411194, -0.7700024843215942, -0.4723722040653229, 0.7240921854972839, 0.39015993475914, 0.4227462410926819, 0.6099779605865479, 0.6426460146903992, -1.6158210039138794, -0.27694350481033325, 1.1052944660186768, 1.2947593927383423, -0.4171395003795624, 0.2859490215778351, -1.2521193027496338, 2.133043050765991, 0.9134650826454163, -0.4187653362751007, 1.5920048952102661, -1.4593263864517212, 0.8325005769729614, 0.2820514738559723, -0.16633829474449158, -1.4721676111221313, 1.4285236597061157, 0.15074773132801056, 0.6213761568069458, -0.7767300605773926, -1.3958531618118286, -0.4232049286365509, -0.06922105699777603, 0.5067950487136841, -1.5383362770080566, -0.1726432889699936, -1.556966781616211, -1.348219633102417, -1.0763487815856934, -1.4000153541564941, -0.042040444910526276, -0.2573997974395752, 0.6227280497550964, 0.06156530976295471, 0.9188748598098755, -0.7669249176979065, 0.4750640094280243], [0.1079612746834755, -1.178126573562622, -1.6124368906021118, -0.4435425102710724, 1.640251636505127, -0.5651054978370667, -1.1549899578094482, 1.2996065616607666, 0.1634681224822998, -0.4159010052680969, -0.9754438996315002, -0.6858494877815247, 0.7455165386199951, 1.9224238395690918, -0.885148823261261, -2.2093441486358643, 1.3034950494766235, 0.4943821132183075, -0.9952561855316162, -0.8178654909133911, -0.0858796164393425, -1.4051443338394165, -0.1079452857375145, 1.3217014074325562, 0.7340992093086243, -0.5437924265861511, 0.24037297070026398, -1.4045060873031616, -1.2862335443496704, 1.2553026676177979, 1.0851073265075684, 0.7146494388580322, -0.5854995846748352, 0.4670238196849823, -0.5394169688224792, -0.37213248014450073, 1.668764591217041, 0.3959773778915405, 1.0851839780807495, -0.7585936784744263, 0.14268024265766144, 0.5737593173980713, 0.6770831942558289, -0.2158583253622055, 0.41819947957992554, 2.8917768001556396, -0.8110544085502625, -0.043142449110746384, 0.8445443511009216, 1.4315037727355957, -0.7222245335578918, 0.6511109471321106, -0.7435749769210815, 1.1839556694030762, 0.4684910476207733, -0.8095899224281311, 0.6809945702552795, 0.14198459684848785, 0.9730215668678284, 0.9940242171287537, 0.019625501707196236, -0.39610257744789124, 0.35362890362739563, -0.3251388967037201], [0.2647498846054077, 0.19207395613193512, 1.2627571821212769, 0.9068818092346191, 1.4093856811523438, 0.16930654644966125, 0.16087622940540314, -0.7886802554130554, 1.6609828472137451, 2.163288116455078, -0.530454158782959, -0.24946023523807526, -2.1794631481170654, -1.3394536972045898, 1.235533595085144, 1.2005093097686768, 2.157397985458374, -0.6892791986465454, 1.9194691181182861, -1.059552550315857, -0.29675954580307007, 2.465264320373535, 1.5198618173599243, 0.46419307589530945, -0.41636988520622253, 1.7253350019454956, -1.4790812730789185, 0.5098555088043213, -0.5258503556251526, -0.15122392773628235, 0.30319875478744507, -1.1645140647888184, -1.7922027111053467, -0.7596058249473572, -0.1999943107366562, 0.11325215548276901, 0.8944586515426636, -2.0548901557922363, -0.20794115960597992, -0.1013580858707428, -0.8111453652381897, 0.49718600511550903, -0.8104513883590698, 0.26826831698417664, 1.8679754734039307, -1.6815084218978882, -0.34717002511024475, 0.6759158968925476, -0.01341166254132986, 0.06730464100837708, 2.0164694786071777, -1.6480381488800049, -1.1637582778930664, -1.0714590549468994, -0.29300612211227417, 1.3094958066940308, -1.2657922506332397, -0.3643020987510681, -0.03359861299395561, 0.7888378500938416, 0.39069655537605286, 1.5351823568344116, -1.2428604364395142, -1.5882703065872192], [-0.03571806102991104, -1.073564887046814, -1.815059781074524, 0.9160414934158325, -0.32734420895576477, 0.11304952204227448, 0.6757737994194031, 0.23722201585769653, -0.24142947793006897, 0.9195752143859863, 1.115309238433838, 0.8240196704864502, -0.4775678217411041, -0.24594224989414215, -0.37144145369529724, 1.102490782737732, 1.6980494260787964, 1.700218677520752, -0.5501829385757446, 0.9958950281143188, 1.042339563369751, -2.3794546127319336, 0.21532344818115234, -0.3338446021080017, 0.7413772344589233, 0.5648232102394104, -0.683840274810791, -1.1319011449813843, 1.0639677047729492, 0.7928522229194641, -1.8548526763916016, -1.3518601655960083, 0.7621836066246033, 0.5114578008651733, -1.1132267713546753, -0.8797764182090759, 1.4902631044387817, 0.8339231610298157, -0.6892172694206238, -0.06872359663248062, -0.11243068426847458, 1.7412610054016113, 0.19486260414123535, 0.2793372869491577, -0.15811365842819214, 0.6353474259376526, 0.03743002936244011, -1.147719383239746, -1.7263360023498535, -2.2282769680023193, 1.5247366428375244, 0.05290522053837776, -1.8114608526229858, 2.764969825744629, -2.0057313442230225, -2.5473906993865967, 0.31401166319847107, 1.0587576627731323, -0.2155531495809555, -1.966373324394226, 1.0882315635681152, -0.3330991268157959, 0.4295791685581207, 1.0532559156417847], [1.3314759731292725, -0.09294534474611282, -1.146122694015503, 0.7311992645263672, -1.3216636180877686, -1.1917195320129395, 1.527401328086853, -2.354694366455078, -0.4006604552268982, 1.4712224006652832, -0.4284450113773346, -0.9941515922546387, -0.06022915244102478, 1.0017900466918945, 1.276907205581665, 0.4696454107761383, -1.7351016998291016, 0.38780540227890015, 0.15682978928089142, 1.4356855154037476, -2.937831163406372, -0.3299386501312256, 0.8619300127029419, 0.7864987850189209, -2.2766144275665283, 0.276741623878479, 0.2597939968109131, -1.7026475667953491, 0.6308276057243347, -0.8152629733085632, -0.111618772149086, 1.2469916343688965, -0.0194556787610054, 1.6148566007614136, -0.7180173397064209, 0.4233763515949249, -0.011768144555389881, -0.7212904691696167, -0.8066465854644775, 0.576575517654419, 1.4207721948623657, 0.5613623857498169, 0.3526446521282196, 1.6739797592163086, -0.7857711315155029, -1.4698143005371094, 0.5505262017250061, 0.038728538900613785, -2.698176622390747, -1.776882529258728, 1.0223416090011597, 2.1203932762145996, 0.7668820023536682, 1.204249620437622, -0.7466263771057129, 0.09431206434965134, -0.3896442949771881, -1.042178988456726, -0.45071473717689514, 0.5224319696426392, 1.1205532550811768, -0.1801530420780182, 0.8683047294616699, -0.09445953369140625], [-1.6519931554794312, 0.43334391713142395, 0.3418201208114624, -1.2352294921875, 1.606021523475647, 2.830514430999756, 0.7551843523979187, -1.5831189155578613, -1.846557855606079, -0.6126821637153625, 0.7625900506973267, 0.3338198959827423, -1.4220203161239624, -0.4317198395729065, 0.7712830305099487, -1.3278396129608154, 0.06977882236242294, -0.8437961935997009, -0.2237006574869156, 1.2278175354003906, -0.3503439128398895, 1.16797935962677, -2.032290458679199, 0.8574094772338867, -0.6093283295631409, 0.549589991569519, -0.07673503458499908, 1.3739466667175293, -1.120009183883667, -2.128448247909546, -0.7365496158599854, 0.1930733621120453, -0.14913791418075562, -0.638110876083374, -1.1031662225723267, -0.9460402727127075, -0.10191325843334198, -1.1173428297042847, 1.645995855331421, -0.8436457514762878, 0.6300775408744812, 0.14969636499881744, 1.3287769556045532, 0.48808351159095764, 0.11859120428562164, 0.9408154487609863, -1.5467908382415771, -1.2999571561813354, 1.0289584398269653, 0.6877345442771912, -1.2817180156707764, -2.348128080368042, 0.4521540403366089, 1.2082269191741943, 1.0620301961898804, 0.674808919429779, -1.0021013021469116, 0.31960710883140564, -0.26228243112564087, 0.16107214987277985, -0.10824555903673172, 0.45211589336395264, -0.4424942433834076, -1.3263286352157593], [0.9523817300796509, 2.43459153175354, 0.4137382209300995, -0.3668740689754486, -0.27183064818382263, 1.5355045795440674, -0.6978620886802673, 0.17790278792381287, 1.1175096035003662, 1.1658399105072021, -0.4236833453178406, -1.258467197418213, 0.6484071612358093, -1.5410795211791992, 0.10892496258020401, 0.2837430238723755, 0.2083519548177719, -1.1359635591506958, 0.9007324576377869, 0.7677565217018127, -1.5918456315994263, 1.002771258354187, -1.0453622341156006, 0.12294268608093262, 0.7248280644416809, 0.05123598873615265, 0.02300250716507435, 0.06794432550668716, 0.35087889432907104, 0.15493658185005188, 1.0689654350280762, -1.486581802368164, 0.09467566013336182, 1.4613969326019287, -2.489609479904175, -0.06927413493394852, -0.4038732647895813, -1.295669674873352, 0.1030103787779808, -0.724543035030365, -2.035855531692505, 0.08315884321928024, 1.403908133506775, 0.32589486241340637, -0.029762543737888336, -0.9672757983207703, 0.06365068256855011, 0.07312550395727158, 0.40272822976112366, -1.1458756923675537, 0.31452661752700806, 0.49173447489738464, 0.2193344682455063, -0.99244225025177, 0.7912675738334656, 0.7695441842079163, 0.9438881874084473, 1.2353854179382324, 0.4238187372684479, -0.4292285740375519, 0.29821500182151794, -0.6337391138076782, 0.204397514462471, -0.33107054233551025]], [[1.3340462446212769, -1.5346704721450806, -1.2776471376419067, 0.1287221759557724, 0.10845351219177246, 0.5282504558563232, 0.17440104484558105, -0.617117166519165, 0.7810393571853638, 0.35267671942710876, -0.6715098023414612, -1.677207112312317, 0.6155229806900024, -0.6862978339195251, -0.5605531334877014, -0.25991320610046387, 0.06353499740362167, -1.6811140775680542, 0.16298602521419525, -1.0846481323242188, -1.1476316452026367, 0.17140968143939972, 1.542977213859558, 0.15923286974430084, 0.38504379987716675, 0.46362993121147156, -0.0758957713842392, -0.648506224155426, -0.5021474361419678, -0.08406799286603928, 0.986061692237854, 1.8659087419509888, 1.5645923614501953, -1.726081132888794, -0.8892182111740112, 0.3838292956352234, 1.025801181793213, 0.5227816104888916, -0.07018211483955383, 1.2382030487060547, 1.2550158500671387, 0.4468619227409363, -1.0867841243743896, -0.4336431622505188, -1.722650170326233, 1.0055434703826904, -1.81892991065979, -0.1625097095966339, -1.61785888671875, 0.2947910726070404, 0.08539474755525589, -0.35695594549179077, -0.39361101388931274, -0.792246401309967, 0.7948977947235107, -1.5883878469467163, 2.266390562057495, 2.1882681846618652, -0.16837561130523682, -0.009880278259515762, 0.2531445324420929, 1.5716978311538696, 0.05378001555800438, -1.5820382833480835], [0.21098098158836365, 0.24085049331188202, 0.23367248475551605, -0.9740201234817505, -1.7144356966018677, -0.812390148639679, 1.394023060798645, -1.2093194723129272, 0.15278778970241547, 0.4472920894622803, 1.5988929271697998, 0.8746757507324219, 1.3572951555252075, 1.109328269958496, -0.661304235458374, 0.5957000851631165, -1.245664358139038, 2.258204936981201, 1.0852891206741333, -1.4209998846054077, -0.9300857186317444, 0.653425395488739, -0.9180313348770142, 0.23817779123783112, 0.6896769404411316, -0.1471548229455948, -2.2510392665863037, -1.740260124206543, 0.691807746887207, -0.6976827383041382, -0.007262126076966524, 0.3225676119327545, -1.2754980325698853, 0.34659844636917114, -1.9381242990493774, -0.2959845960140228, 0.5817258954048157, 1.9521313905715942, 1.3140645027160645, 1.974884033203125, 1.299557089805603, 0.8265095949172974, 0.2727760076522827, 1.7288587093353271, -1.0197010040283203, -0.8883903622627258, -0.9943159818649292, -1.197710633277893, -1.0566301345825195, 0.21752868592739105, -1.4967166185379028, -0.2534613013267517, -0.2641121447086334, 0.1319870501756668, -0.5063642859458923, -0.7350525259971619, -0.2908066511154175, 0.7410039901733398, 0.23109814524650574, 0.05830013379454613, -0.3566206991672516, 1.1751350164413452, -1.4448579549789429, -2.2373206615448], [-1.81161367893219, 1.1606078147888184, 0.0589078851044178, -1.2420765161514282, 0.6462399363517761, 0.4838264584541321, 0.4488161504268646, 0.018359245732426643, -0.26613649725914, 1.3747507333755493, 0.29387837648391724, 0.011442327871918678, -0.7330398559570312, 0.13389788568019867, 0.5513520836830139, -1.4916012287139893, 0.4338633120059967, -0.38700631260871887, -0.9239317178726196, 0.5368218421936035, 2.247563362121582, -1.1154927015304565, -0.6039007306098938, -1.8673217296600342, 0.9551874995231628, 0.661564290523529, -0.8017662167549133, 1.4892102479934692, 0.8410003185272217, -0.07781556993722916, 0.441607803106308, 0.2877582609653473, 1.3217461109161377, -1.9957720041275024, 1.3739194869995117, -1.4815560579299927, 1.519698977470398, 0.555057168006897, 0.14802183210849762, -0.8140326142311096, -0.22385726869106293, 0.4982047975063324, 0.7954748272895813, 1.5949511528015137, -0.5223003625869751, -0.5655446648597717, -0.35807284712791443, -0.3417026400566101, 0.20218636095523834, -0.6382884979248047, 0.4165228307247162, -1.3043243885040283, -0.547444224357605, 0.5052507519721985, -1.4276431798934937, -0.791780948638916, 0.44584691524505615, -0.04314348101615906, -0.3885185122489929, 1.6931626796722412, -0.17536792159080505, 1.0761502981185913, 0.15714715421199799, -0.8176491260528564], [-0.41951075196266174, 0.38985002040863037, 0.5920450091362, -1.0725243091583252, -0.3523346483707428, 1.2688140869140625, 0.28882208466529846, -1.4626110792160034, -0.11964297294616699, 0.17225807905197144, -0.9195331931114197, -1.2197753190994263, 0.3635430634021759, -0.21246905624866486, 1.8911337852478027, -1.0999797582626343, 0.1906764954328537, 0.4076322019100189, -1.2097251415252686, 0.50749671459198, -0.7401384711265564, 0.8895146250724792, -1.5856051445007324, -0.3732719123363495, 0.12643831968307495, 0.714486837387085, -1.043555498123169, -1.603877067565918, 0.6515832543373108, -0.44418251514434814, 0.725369393825531, -1.3824952840805054, -0.9134519100189209, -0.994240403175354, -0.87492835521698, 0.4787808358669281, -0.8759537935256958, 0.6093623638153076, -0.8604093790054321, -1.2868938446044922, 0.3006233274936676, 0.37618133425712585, 2.0287671089172363, -2.600975751876831, -0.8406355977058411, 0.3101212680339813, -0.28808820247650146, 1.7563748359680176, -0.4817017912864685, 0.2488914579153061, -0.536908745765686, -2.164926528930664, -2.7600533962249756, -1.6843734979629517, -2.469153881072998, 0.5822861790657043, -0.11048127710819244, -1.9656671285629272, 0.23983389139175415, 1.2580015659332275, -0.7392769455909729, -1.2642344236373901, -0.8005959391593933, -0.0036191255785524845], [1.1959726810455322, 0.565572202205658, 0.44971656799316406, 0.09543583542108536, -0.6542319059371948, -1.756625771522522, 0.35055941343307495, 0.3217517137527466, -0.754188597202301, -0.19716797769069672, -0.9955018162727356, -1.0251888036727905, 0.6317073702812195, -0.7034367322921753, 0.011991696432232857, -0.4997899830341339, -0.3730678856372833, 1.1648684740066528, -2.0422894954681396, -0.5738495588302612, 2.188246726989746, 0.3143080472946167, 0.2191867083311081, 0.7311874032020569, 1.145680546760559, 0.4063992202281952, -0.6386862397193909, -0.24890801310539246, 1.692233920097351, -0.6711574196815491, -1.6659809350967407, 0.13228286802768707, 0.41366660594940186, 0.45507943630218506, 0.15737532079219818, -0.8172154426574707, 1.289679765701294, 1.9054129123687744, -0.9669414758682251, 1.736833095550537, 0.5594533681869507, -1.1902936697006226, 0.04632464051246643, 0.18741483986377716, 0.7083494663238525, 0.485451340675354, 0.8939657211303711, 0.33009618520736694, 1.2471810579299927, -0.17968526482582092, 0.7403998374938965, -1.5131500959396362, -0.07777278125286102, 2.002448320388794, -1.0774863958358765, 2.9384682178497314, 2.466109275817871, -0.04586417227983475, -1.6524152755737305, 0.3369765281677246, -1.5479536056518555, 1.0719578266143799, -1.0868351459503174, -0.21420563757419586], [-1.8068833351135254, -0.30993151664733887, -0.06765174865722656, 0.4957166016101837, 1.0847066640853882, 1.6154234409332275, 0.15538626909255981, -0.8665551543235779, 0.9136213064193726, -1.3917632102966309, 1.0826153755187988, -0.930596113204956, -0.25413280725479126, -0.952540397644043, -1.4854835271835327, -0.6652039289474487, 0.22094596922397614, -1.3744066953659058, 1.199278473854065, -0.49366679787635803, -0.1952829509973526, -0.439778208732605, 0.011491550132632256, 0.3002108633518219, 0.4214594066143036, -0.24837364256381989, 2.284590482711792, 1.0506917238235474, 0.6144401431083679, -0.14106470346450806, -0.3301955759525299, 1.4958617687225342, -1.3265373706817627, -0.5747475624084473, -0.48009398579597473, 0.9503005743026733, 0.7457674741744995, 1.2579175233840942, -0.5754433274269104, 0.5606024265289307, 0.29328131675720215, 0.1717132180929184, 0.6180459856987, 1.126302719116211, 0.5262008905410767, 0.7493679523468018, -1.0848180055618286, 0.2372211366891861, -0.7182983756065369, -0.12202909588813782, 0.7903457880020142, -2.7705063819885254, -0.7041110992431641, 1.0670990943908691, 0.9895842671394348, -1.3876969814300537, -0.738437294960022, -1.6065428256988525, 1.1997151374816895, -0.28850501775741577, -0.12408001720905304, -1.4494863748550415, -0.2846980094909668, -0.5099372267723083], [-2.7688167095184326, -1.5255074501037598, -2.981712818145752, -0.345665842294693, -0.4203084409236908, 1.4699851274490356, -2.4547488689422607, 1.3023018836975098, 0.3893396258354187, 0.717572033405304, -0.18033918738365173, -0.6192570328712463, 1.4030303955078125, -1.691902756690979, 1.2195693254470825, 1.0319222211837769, 0.5066291093826294, -0.23757584393024445, 0.764606237411499, 0.38164883852005005, -1.4347765445709229, 0.28124332427978516, 0.03216835856437683, 0.47405678033828735, 0.9497878551483154, 1.0308243036270142, 0.4952124059200287, -0.16008701920509338, -0.06109614297747612, -0.5112041234970093, -1.2474229335784912, 0.633922278881073, 0.3996618986129761, 0.42950373888015747, 0.564572811126709, 0.1912776231765747, -0.7335266470909119, -0.3972913920879364, 0.14653639495372772, 1.6647013425827026, 0.6974106431007385, 0.4111306369304657, 0.08065754175186157, 0.11127801239490509, -0.03214649856090546, -0.957160234451294, 2.353066921234131, -1.2476001977920532, 0.19646140933036804, -0.7034879922866821, -0.8804206252098083, 2.0259010791778564, 0.3334009051322937, -1.869641661643982, 1.0814388990402222, 2.511021614074707, 0.7050632834434509, -1.6869211196899414, 0.8198149800300598, -0.3135402798652649, 0.7766656875610352, 0.9759955406188965, 0.031153056770563126, -1.301391363143921], [0.002882617060095072, -0.864704430103302, 1.0666483640670776, 0.14152075350284576, -0.3517150282859802, -1.7494832277297974, -1.7143827676773071, 0.16470597684383392, 0.6715195178985596, 0.4938163459300995, 1.984713077545166, 0.7521482706069946, -1.3559921979904175, 2.002753734588623, 1.3838257789611816, -0.5151078104972839, -0.42506593465805054, 1.9816545248031616, 1.0822254419326782, 1.1243948936462402, -1.0778697729110718, -0.3094760477542877, -0.993646502494812, -0.6530218124389648, -0.5713019371032715, 0.6459251046180725, 0.7571723461151123, -0.5417315363883972, 1.9603278636932373, 0.05084004998207092, -0.19311270117759705, 0.648151695728302, -2.0143375396728516, 0.18532878160476685, 0.38367971777915955, -0.6179466247558594, 0.8181671500205994, 0.7857585549354553, 0.9846933484077454, 0.971111536026001, 1.632079005241394, 0.5985841751098633, -0.5407556891441345, 0.05072811245918274, 0.428242564201355, -0.3267712891101837, 1.4248261451721191, 1.6348756551742554, -0.9736685752868652, -0.3765887916088104, 0.10363530367612839, -0.0993359312415123, -1.3292641639709473, 1.08529531955719, 1.2003169059753418, 1.078646183013916, -1.8283263444900513, 0.10772042721509933, -1.8693633079528809, 0.07242367416620255, 1.5743521451950073, 0.06698082387447357, -0.9772310256958008, -1.1192692518234253], [0.7059117555618286, 1.3650728464126587, 0.28618717193603516, 0.5088871121406555, 0.8336738348007202, 0.23915590345859528, -0.26974424719810486, 0.8184224963188171, 1.505191683769226, -0.40397271513938904, 1.1331872940063477, -1.270864725112915, -1.5615066289901733, 0.12111131846904755, 1.1249805688858032, -0.21252119541168213, 0.34929338097572327, 0.7565247416496277, 0.20155197381973267, 0.6535778641700745, 0.964544415473938, 0.7209293842315674, -1.6137663125991821, -0.3911270201206207, -0.04526209831237793, 1.78336501121521, 0.40928569436073303, 0.8372264504432678, 0.8326970338821411, -0.24878205358982086, 0.861754298210144, 0.9152221083641052, 1.0039196014404297, -1.9283359050750732, 1.1045249700546265, 0.11166827380657196, -0.7552402019500732, 0.05986257269978523, 0.5467034578323364, 0.33502137660980225, 0.41275614500045776, -1.6374751329421997, -0.6176034212112427, 1.6006019115447998, 0.7669300436973572, 0.3531596064567566, -1.3154540061950684, 0.09189524501562119, 1.0507184267044067, 0.378611296415329, -0.4717326760292053, 0.2883109152317047, 0.5816047787666321, -0.8719629645347595, -0.9425174593925476, 1.3953089714050293, -0.5076507329940796, 0.42582932114601135, -1.7403837442398071, 1.2024545669555664, -0.6974551677703857, -0.38714468479156494, -0.007289333269000053, -0.6776732802391052], [0.8761478066444397, 0.9179368615150452, -0.3892427086830139, 0.38819608092308044, -0.24317553639411926, 1.5437973737716675, -0.0409938208758831, 1.3991304636001587, 0.26970353722572327, 1.1454713344573975, 0.9003570675849915, 0.9119595289230347, -0.09370800107717514, -0.23791080713272095, -1.8611705303192139, 0.3939945101737976, 0.5758221745491028, -1.3005554676055908, -1.9832334518432617, 0.01373556163161993, -0.7591463923454285, 0.5097705125808716, 3.1918563842773438, -0.03204956278204918, 0.8679468035697937, 0.8759178519248962, -0.888192355632782, -1.0698561668395996, -0.37862536311149597, 1.4798457622528076, 0.16385719180107117, 2.511021852493286, 0.7484445571899414, 0.2630501091480255, -0.8672106266021729, 1.0716248750686646, -2.2984535694122314, -0.03921768441796303, 0.08124884217977524, 2.1735799312591553, 1.5197604894638062, 0.13122470676898956, -0.4808948040008545, 1.1237748861312866, 0.12382945418357849, -0.2645317018032074, 2.6415815353393555, -0.9568057060241699, 0.8518780469894409, -0.052272334694862366, 1.7058801651000977, -1.5874673128128052, -1.5836211442947388, -0.6561729311943054, -0.7888287901878357, 0.071555495262146, -1.1951708793640137, 0.25761300325393677, -0.6694763898849487, 0.020138133317232132, 1.9906878471374512, -0.8811867237091064, 0.032434165477752686, -0.5063203573226929], [0.6838266849517822, 0.4255801737308502, -0.0633796900510788, -1.1423860788345337, 0.3318638503551483, 0.28248080611228943, 0.14936061203479767, 0.5419828295707703, 2.39859938621521, -1.5377508401870728, -1.0753540992736816, -2.3561952114105225, -0.4894341230392456, -0.40394553542137146, -2.0389368534088135, 0.5671156048774719, -0.6018934845924377, 1.7712271213531494, -0.39596670866012573, 0.2881534695625305, -0.2742651402950287, -1.4285588264465332, 1.4593515396118164, 0.7640348672866821, -1.3513542413711548, -0.53242027759552, 1.7638840675354004, -1.6109514236450195, -1.3000797033309937, -0.9603842496871948, -0.9679659008979797, -0.9549384117126465, 0.47863778471946716, -0.4895903468132019, -1.1977870464324951, -0.6100513339042664, 1.9848742485046387, 1.8488117456436157, -0.024742895737290382, -0.397635281085968, 0.15958072245121002, -1.809446096420288, -0.3610536754131317, -1.3125550746917725, 0.5120975375175476, -1.0206011533737183, -0.24239253997802734, -0.23221011459827423, -0.07764500379562378, 0.32677483558654785, 1.9331508874893188, -0.8018123507499695, -0.1304389238357544, 0.7450947761535645, 0.29860660433769226, 0.8844764232635498, 0.9891070127487183, -1.2468699216842651, 0.5373407602310181, -0.3454706370830536, 0.42865875363349915, 1.3253849744796753, 0.47495388984680176, 0.9957704544067383], [-1.663960576057434, 0.4914518892765045, -0.1215878576040268, 0.1354624480009079, -0.29386216402053833, 0.5332263112068176, -0.22131098806858063, -1.6884405612945557, -0.23604445159435272, -0.5627837777137756, -0.520246148109436, 0.297910213470459, 1.3924959897994995, 0.14897790551185608, 3.2605807781219482, 1.429301381111145, -0.38803690671920776, -0.7367172241210938, 1.7333564758300781, 1.3557054996490479, 1.4565794467926025, 1.6431108713150024, 1.2034859657287598, -0.5533969402313232, -0.5922225117683411, -0.5556549429893494, -0.5952415466308594, -0.8442329168319702, -1.4185744524002075, -2.1255762577056885, -0.24991244077682495, 0.3844072222709656, 1.7590711116790771, 1.2489346265792847, -1.1197651624679565, -0.21775038540363312, 0.36170560121536255, -0.5576170086860657, -1.811985969543457, 0.29704222083091736, 0.06835416704416275, 0.18753619492053986, 0.2992722690105438, -0.9011162519454956, 0.48546966910362244, 0.4114774465560913, -0.25288155674934387, -0.9028388261795044, -1.0903247594833374, 0.48575863242149353, 0.2023448944091797, -1.3559317588806152, 0.08901496231555939, 0.41917628049850464, -2.0914831161499023, 0.20074203610420227, 0.04114663600921631, -2.1080162525177, 0.27016398310661316, -0.14700336754322052, -0.25159579515457153, -1.2014049291610718, -0.4761340022087097, 0.39365988969802856], [0.16246488690376282, -1.53462815284729, 0.19442413747310638, -0.36049413681030273, 1.2785930633544922, 1.882266879081726, -1.898266077041626, 1.1291882991790771, 0.7183877229690552, 0.9184252619743347, 1.3555876016616821, -0.9627693295478821, 2.3679888248443604, 2.805894374847412, -1.1261742115020752, 1.8871532678604126, 0.6928099989891052, 1.7143551111221313, 1.1959474086761475, 0.2656201422214508, 0.36159852147102356, -0.21622073650360107, -0.1804133504629135, -0.22679713368415833, -0.1609942764043808, 1.558529257774353, 1.1254196166992188, -1.6394327878952026, -0.7811004519462585, 2.28682804107666, 0.463192880153656, -0.0014380449429154396, 0.5390143394470215, -0.25313180685043335, 1.1938717365264893, 0.2632448375225067, 0.23393748700618744, 0.28357020020484924, 1.1480839252471924, 0.5683402419090271, -1.520421028137207, 0.15801052749156952, -1.9505385160446167, 0.9603298306465149, -0.17339468002319336, -1.9048455953598022, -0.5345818996429443, 0.6286154389381409, -0.6464714407920837, -0.2348410040140152, 0.8377131223678589, 2.19988751411438, 0.3175088167190552, 1.3408294916152954, -0.5028790831565857, 0.05547294765710831, -0.35822173953056335, 0.14928652346134186, 2.4421069622039795, -0.4144344925880432, 0.7964881658554077, 0.6770954728126526, -0.07665497064590454, -1.306041955947876], [-0.999545156955719, -0.21853794157505035, 1.7537846565246582, 1.0589568614959717, -0.44105085730552673, -0.0831449031829834, -1.6981562376022339, -0.8393661975860596, 0.4890024662017822, -0.7549655437469482, -0.3033987283706665, 0.4937599301338196, 0.7913530468940735, 2.170301914215088, 0.15170833468437195, -0.6426019072532654, 0.4161474108695984, 1.6073423624038696, 0.4746098518371582, 0.5923330187797546, -0.5841712355613708, -0.07413838803768158, 0.712123453617096, -0.8927935361862183, 0.34227070212364197, 1.7610273361206055, 0.4661853611469269, -0.16913814842700958, -0.016878660768270493, 1.4530144929885864, -0.3085763454437256, -1.5659085512161255, -1.5489190816879272, 1.1387463808059692, 0.951460063457489, 0.7105292677879333, 0.9596329927444458, 0.397687703371048, 0.6479960083961487, -0.5224999785423279, -0.9608180522918701, 1.4919177293777466, -1.22390878200531, 1.8308392763137817, 1.111789345741272, 0.9957287311553955, 1.2631624937057495, -1.119734287261963, -0.5587321519851685, -0.2767731547355652, 1.4767022132873535, -0.3281194269657135, 0.33851611614227295, -0.05591706559062004, -0.10544466972351074, -1.3003122806549072, 0.40424713492393494, 0.34716033935546875, 0.0005554936942644417, 0.027857786044478416, 0.7195689678192139, -0.12106563150882721, 0.7784875631332397, -0.26750680804252625], [0.34078869223594666, -1.1278860569000244, -0.34431684017181396, 0.33375272154808044, -0.23218201100826263, -0.38088351488113403, -0.19302481412887573, -0.67910236120224, 0.333161860704422, 1.0801100730895996, 1.3827382326126099, -0.12333495169878006, -0.7114970088005066, -0.5028367042541504, -0.016308678314089775, 0.28417858481407166, -0.06915165483951569, 0.7062626481056213, -0.4138045012950897, 0.26622676849365234, -0.07592765986919403, -1.0281035900115967, -1.0795081853866577, 1.396469235420227, 1.328722357749939, 1.4061124324798584, 2.072958469390869, 1.1952983140945435, 1.5474448204040527, -0.6548864841461182, 0.10407040268182755, -0.5774788856506348, 0.5669749975204468, -2.0089166164398193, -0.5382843017578125, 0.14618465304374695, -1.8783812522888184, 1.3326727151870728, -1.0008397102355957, 0.3384149968624115, -1.254077672958374, -0.23190315067768097, 0.8438881039619446, 0.7858874201774597, -1.7356771230697632, -0.14569906890392303, -0.0969250500202179, -0.31512632966041565, 0.2610941231250763, 1.4403445720672607, 0.04331548139452934, 1.3358699083328247, -1.1144789457321167, 1.2040420770645142, -0.21883758902549744, -0.7105646729469299, 0.3006296157836914, -0.2047630399465561, 0.21019475162029266, 0.5008648037910461, -0.37860581278800964, -0.827191174030304, 0.6545153260231018, 1.8619004487991333], [0.6844220161437988, -0.601478099822998, -1.9354526996612549, 0.7072392702102661, 2.4177730083465576, -0.1741953343153, 0.017825864255428314, -1.924646019935608, -0.4473404884338379, -0.3469984829425812, -0.22206801176071167, 1.3491653203964233, -0.7252739071846008, -1.490857720375061, 0.5173566937446594, -0.46234366297721863, -0.8244308829307556, -0.351791650056839, -0.5212371945381165, -0.2439795583486557, 0.6454957127571106, 0.20403780043125153, -1.3567415475845337, 0.1523982733488083, 0.568221926689148, -0.9791455268859863, 2.0667684078216553, 1.6556988954544067, -0.05611753091216087, -1.7267931699752808, -1.6222221851348877, -0.044826630502939224, 0.4708004891872406, 0.8068974614143372, -0.6192606091499329, 0.4646504521369934, 0.17278769612312317, 0.8069787621498108, 0.3223337233066559, -1.2414441108703613, 1.0134668350219727, 0.3088182806968689, 0.9873359799385071, -0.7084733843803406, 0.7750965356826782, -0.7780454754829407, -0.10328322649002075, -0.6969770789146423, -0.36092156171798706, 0.2765319049358368, -1.226103663444519, -1.1618893146514893, 1.3858983516693115, -0.8963788747787476, 0.5734605193138123, -1.8046848773956299, 0.4474916458129883, 0.8282909393310547, -0.5270226001739502, -0.4593992531299591, -0.5960585474967957, 0.7562808990478516, 1.54812490940094, 0.6049963235855103]]], [[[-0.0035043407697230577, -1.965027093887329, 1.2606388330459595, 1.3275344371795654, 1.650688648223877, 0.1143346056342125, -0.9443110227584839, 0.5807506442070007, 0.5995644927024841, -0.2808294892311096, -0.8244168162345886, 0.6921457648277283, 0.6840089559555054, -0.8769158124923706, 0.2884831726551056, -0.3373187184333801, -0.9157685041427612, -0.31728702783584595, -0.06904439628124237, 1.4468048810958862, -1.6144863367080688, 0.34549862146377563, -0.025299230590462685, 0.648374080657959, -1.056593418121338, -1.5809987783432007, 2.0338704586029053, -1.170409083366394, -0.9321979880332947, -1.7077937126159668, 0.9405695796012878, 0.1184430792927742, -1.8205853700637817, 0.08880840241909027, 2.637526273727417, -0.6606269478797913, 0.3151310682296753, -0.05196134373545647, 0.11958079040050507, -2.024534225463867, -0.9178933501243591, -1.8885245323181152, 0.09228202700614929, 1.8156194686889648, -1.1930264234542847, 0.507929265499115, 0.9763310551643372, 0.6355945467948914, -0.10421761870384216, -0.11542994529008865, -0.6109566688537598, 2.681284189224243, 0.2178967297077179, 0.29970431327819824, 0.14330139756202698, 1.074090838432312, 0.5359312891960144, 0.507987916469574, 2.487351417541504, -0.10783462226390839, 0.9274989366531372, 1.0424141883850098, -0.9954321980476379, -0.6379263997077942], [1.3108704090118408, -1.2243235111236572, 1.6122045516967773, -0.24193896353244781, -0.36194270849227905, -0.5510807633399963, 0.6233330965042114, -1.3176060914993286, -0.3624251186847687, 0.004635340068489313, 1.1928555965423584, 0.7469760775566101, 1.0801165103912354, -0.22155746817588806, 0.013117793016135693, -0.13768082857131958, 1.032599925994873, 0.5614415407180786, 0.8287119269371033, 0.601574718952179, 0.01698227971792221, 0.21819399297237396, 1.5766119956970215, 0.39284223318099976, 1.6074762344360352, -0.8411681056022644, -0.20298825204372406, -0.29710397124290466, -2.039069890975952, 0.6379614472389221, -1.0197511911392212, 0.23160827159881592, 0.5271843671798706, 0.3726716935634613, 0.22476615011692047, -0.09135009348392487, 0.7333868741989136, -0.4734139144420624, 1.1535216569900513, 0.15058328211307526, -0.8088114261627197, -1.2447925806045532, 1.5490068197250366, 0.2821041941642761, -1.1111292839050293, 0.020790893584489822, 0.518920361995697, 1.4143751859664917, -2.093676805496216, -2.9201807975769043, 0.8187131285667419, -0.5364447236061096, -1.880205512046814, -0.746749222278595, -2.0526070594787598, -0.6235450506210327, -0.527405858039856, -0.07565044611692429, 0.7587063312530518, -2.0147511959075928, -2.4645676612854004, -0.36327555775642395, -0.6343874335289001, 1.0505356788635254], [-0.4085988402366638, 1.5051857233047485, -0.27551326155662537, -0.11363039165735245, 0.4451833665370941, 0.7445854544639587, 1.666689157485962, 0.020694434642791748, -0.08455429971218109, -0.36859259009361267, 1.351860523223877, -1.3851603269577026, 1.0551657676696777, -1.014696478843689, -0.5381258130073547, 0.9790154099464417, -0.16129888594150543, -1.1924182176589966, -0.1367144137620926, -0.4511609375476837, 0.004396640229970217, 0.18571221828460693, 0.9592229723930359, 0.3871864974498749, -1.0939035415649414, -1.6969704627990723, -0.5550876259803772, 1.3889755010604858, 0.3473759889602661, 1.8121811151504517, -0.9187291264533997, -0.3163723945617676, -1.526667594909668, 0.37734365463256836, 0.678606390953064, 1.1964713335037231, -1.3893802165985107, -1.459288477897644, 0.5435578227043152, -0.3119731843471527, -1.8747472763061523, -0.42974740266799927, -0.06516040861606598, -1.6971529722213745, -0.6920297145843506, 0.08180849999189377, -0.5225794911384583, 0.199692502617836, 0.14400488138198853, -0.8540441393852234, 0.2855721414089203, -1.1976066827774048, -1.134508728981018, 1.2406741380691528, -0.7470465898513794, 1.319879174232483, 0.2070544958114624, 1.477077603340149, 0.13988742232322693, 0.812926709651947, -0.42501407861709595, 1.647939682006836, 1.7446939945220947, -0.6056336760520935], [0.7352038025856018, -0.07419486343860626, 0.9678552746772766, -1.4382823705673218, 0.11575613915920258, 0.914422869682312, 0.5232232213020325, -0.6192014217376709, -2.325732469558716, -0.27987203001976013, -0.38225990533828735, -0.6179214715957642, -0.41666659712791443, 1.4822710752487183, 0.6247348189353943, -0.8270338177680969, 1.4600428342819214, -1.1892907619476318, -0.2851915955543518, 1.3389583826065063, 1.1434415578842163, -0.32481494545936584, 1.7702327966690063, 0.7706169486045837, -0.7946378588676453, 2.0185766220092773, -0.602687418460846, 0.9978245496749878, -1.0006699562072754, -2.2639753818511963, 0.5127293467521667, -1.1796588897705078, 2.2652649879455566, -0.1373605728149414, 1.3473178148269653, 0.8474147319793701, 1.7817434072494507, 0.1817164570093155, 2.3410401344299316, -0.22765956819057465, -0.3439977765083313, -0.5381550192832947, -0.5182314515113831, 1.2690354585647583, -1.5594855546951294, -2.7472481727600098, -1.752937912940979, -0.8820528984069824, -2.0867507457733154, -0.7046137452125549, -1.2275532484054565, 0.5269913673400879, 0.33500802516937256, -0.2139955759048462, 1.0709097385406494, 0.2052765041589737, -1.353052020072937, 2.337608814239502, 1.100825548171997, -0.18268240988254547, -0.16378694772720337, 0.4060560464859009, -2.3558926582336426, -1.5721946954727173], [0.21860529482364655, 1.294670820236206, 1.7935564517974854, 0.4595530927181244, 1.5253901481628418, 0.12211234867572784, 0.5557069182395935, -0.36668288707733154, -1.405814290046692, -0.8357160687446594, 0.6345207095146179, -0.5779303908348083, -1.4407018423080444, 0.24924376606941223, 2.6847033500671387, 1.634706735610962, -1.5375151634216309, 1.292893409729004, 1.348042368888855, -0.03514249250292778, 0.8104054927825928, 0.6978165507316589, 1.0887322425842285, -0.17127588391304016, -0.14276906847953796, -1.0657620429992676, -0.22972320020198822, 0.6872656345367432, 0.6928166747093201, 0.059498682618141174, 0.02373761683702469, -0.043640196323394775, 0.8530226349830627, -0.8121101260185242, -0.5512983202934265, -1.1079137325286865, -0.9658628702163696, 1.7665257453918457, -0.43132254481315613, 1.1532530784606934, -1.5915825366973877, 0.02357509918510914, 0.015646012499928474, 1.1336650848388672, -0.8343954682350159, -0.3538512885570526, 0.23571287095546722, -0.5896095037460327, 1.6503719091415405, -1.2182036638259888, 1.366613507270813, -1.805932641029358, 2.755619764328003, -0.12843570113182068, 0.9187455773353577, 0.13441777229309082, 0.5017005801200867, 0.12639328837394714, 0.7767465114593506, -1.3048646450042725, -0.06508409231901169, -0.3858901560306549, 0.4969216287136078, -2.1017048358917236], [-0.5584258437156677, -0.6280712485313416, 1.7449487447738647, 1.1092127561569214, 0.8348028659820557, 0.21892404556274414, 0.17392221093177795, -1.2466033697128296, -0.9749394655227661, 0.6020357012748718, 0.8108869194984436, -0.8538556694984436, 0.831892728805542, -0.8950169086456299, 0.26600778102874756, -0.3237507939338684, 0.6455522775650024, 0.043322958052158356, -0.2996077835559845, 0.5343952775001526, 0.06404734402894974, 0.453594446182251, -0.27835556864738464, 2.454411745071411, -1.2003742456436157, 0.9886402487754822, -0.2810100317001343, 1.1159319877624512, -0.32442083954811096, 0.9258794784545898, 2.6694304943084717, -0.9051360487937927, -1.3189738988876343, 1.4397461414337158, -0.4895765781402588, -2.1216297149658203, -0.21184702217578888, -1.3884857892990112, -1.090085744857788, 0.6258749961853027, -0.035761479288339615, 0.8246563673019409, -0.29264700412750244, 0.17265209555625916, -0.4870496094226837, 0.4633949100971222, 1.2396982908248901, -0.3398243486881256, 2.372955799102783, -0.6897061467170715, -1.7849056720733643, -0.654575526714325, 1.8513530492782593, -0.19403067231178284, 0.4848315119743347, 2.097362995147705, 0.42836859822273254, 0.04190254211425781, -1.173960566520691, -1.3529947996139526, -0.016448305919766426, 1.1547645330429077, -1.4383518695831299, -0.27889400720596313], [-1.500440239906311, 1.052250623703003, -0.027472687885165215, -0.714005708694458, -0.27775201201438904, 2.2283308506011963, 0.08555468171834946, 2.673696279525757, -0.1659928262233734, -0.7391124963760376, 0.2966899275779724, 1.8984678983688354, -0.3331044316291809, 0.23505660891532898, 2.1185567378997803, 1.3951925039291382, 0.6042458415031433, -0.2299034148454666, -1.6872326135635376, -0.5239519476890564, 0.3973168432712555, -0.40602046251296997, 0.17542439699172974, -0.9425041675567627, -0.05401301383972168, -0.12780453264713287, 1.2226272821426392, 0.3629244863986969, 0.36897799372673035, -0.2512767016887665, 0.13207247853279114, 0.5541733503341675, 0.19708584249019623, -0.046521902084350586, 0.5842717885971069, 0.4957689642906189, 0.8650994896888733, 0.06105181202292442, -1.2294855117797852, 0.2294754534959793, -0.2106582522392273, 1.1137912273406982, -1.0727664232254028, -0.5790515542030334, 0.1538182646036148, 1.116265058517456, 0.2704755961894989, 0.5311604738235474, 0.4431697428226471, 0.4406416416168213, 1.8105542659759521, 0.2080727070569992, -0.6141895651817322, 0.558457612991333, 0.3727478086948395, 0.006733720190823078, -0.7478307485580444, 1.627138614654541, 0.06353633105754852, -1.63107168674469, -0.4739901125431061, -1.3001041412353516, -1.1140085458755493, -0.49231410026550293], [1.2935841083526611, 1.6327214241027832, -0.17493687570095062, -0.9495290517807007, -0.05027291551232338, 0.27789318561553955, 0.10696835815906525, -0.125026136636734, -0.29195520281791687, 0.17017874121665955, -1.0337650775909424, -0.1869879961013794, -0.42413821816444397, 0.7027121782302856, 0.9934093952178955, -2.4579954147338867, 1.4690139293670654, 1.327958583831787, -0.7080667614936829, -1.6241555213928223, 1.2779942750930786, -1.2115126848220825, -0.020419185981154442, 0.7266874313354492, -0.8489572405815125, 0.6161370277404785, 0.3671504557132721, -0.8918173909187317, -0.7419595122337341, -1.2309253215789795, 0.7832388281822205, 0.07540316879749298, 0.5110421180725098, 0.4702819287776947, -1.3827632665634155, 0.12992632389068604, 0.6643840074539185, -0.3293497562408447, -0.8900421261787415, 1.0948388576507568, 0.5016492605209351, -0.6012323498725891, 0.5583955645561218, 1.179351568222046, -1.1588631868362427, -4.023889064788818, 1.8350059986114502, 0.6550946831703186, 0.21091464161872864, -0.0518912710249424, -1.8308829069137573, -0.4528232216835022, -1.0607683658599854, 0.33470866084098816, 0.5045276880264282, -0.8931831121444702, -1.231337308883667, -1.2264748811721802, 1.0283280611038208, -0.6796553134918213, 1.7217514514923096, 0.13155150413513184, 0.4654778838157654, 1.2314817905426025], [0.18191266059875488, -0.21106921136379242, 1.0503309965133667, -1.2475810050964355, -0.48578429222106934, -0.16410788893699646, -0.0806041955947876, -0.049258239567279816, 0.29427430033683777, -0.24506445229053497, 0.1492432951927185, 1.8550689220428467, -1.0836601257324219, 0.24131999909877777, -0.10100027173757553, -0.21340182423591614, -0.7914363741874695, -1.3214490413665771, 2.1622352600097656, -0.7098560333251953, 0.37237247824668884, -0.3077773451805115, -0.4288070499897003, 0.7096017599105835, -1.1800556182861328, 0.5154109597206116, -0.6091048717498779, 0.12328164279460907, 1.9343916177749634, -0.0008725334191694856, -1.3183355331420898, -1.2830191850662231, 0.8641238808631897, 0.32158249616622925, 1.0246614217758179, 0.2073606550693512, 0.08407779783010483, -0.7524262070655823, -1.1675668954849243, 0.29821211099624634, 0.6378746628761292, -0.003945319913327694, 0.5901219844818115, -0.9827567934989929, -1.127629041671753, 1.71513032913208, 0.9394354224205017, -0.12323293089866638, 0.22206470370292664, -0.6744037866592407, 1.2477422952651978, -0.29218029975891113, -0.30580076575279236, -0.4756348133087158, -0.6138166189193726, 0.028973689302802086, 0.01564689539372921, -0.19719842076301575, 0.37578627467155457, 0.38440632820129395, -0.2030658721923828, 0.05218358710408211, -0.5939542055130005, 0.48600512742996216], [0.24865153431892395, 1.7413270473480225, 0.34508419036865234, -1.021872639656067, 0.5810531377792358, 0.10200836509466171, -0.3017948567867279, 2.400177001953125, 0.9265992641448975, -0.7282293438911438, 1.6959800720214844, 0.27557694911956787, 0.42530304193496704, -1.5694122314453125, 0.7692527770996094, -0.003852797206491232, -0.4372217357158661, 1.128618836402893, -1.0123820304870605, 0.48110368847846985, 0.34016016125679016, -1.0069066286087036, 1.2656362056732178, -0.40786683559417725, -0.5064010620117188, 0.1328715682029724, -0.35833659768104553, -0.9859869480133057, -0.341802716255188, -0.28920021653175354, -1.2996082305908203, -0.12369677424430847, 0.923089861869812, 1.1258658170700073, 0.5171814560890198, 0.651850163936615, 0.7999390959739685, 0.09892889112234116, 1.4385898113250732, -0.4936028718948364, -0.5689167380332947, 0.20005613565444946, 0.7159045338630676, -0.785685122013092, -0.14419785141944885, 0.9565882086753845, 1.0417159795761108, 1.3774231672286987, 1.4438204765319824, 0.16782814264297485, -0.7852239012718201, 0.27899783849716187, 0.9586000442504883, -1.6538934707641602, -2.0375709533691406, 1.3188822269439697, -0.5543436408042908, 0.1612129956483841, 0.7095350027084351, 0.35850071907043457, 0.5037276148796082, -0.8662235736846924, 1.5377860069274902, 1.862043023109436], [0.016113029792904854, 0.7864935398101807, -1.9360407590866089, 1.1230173110961914, 0.5652502775192261, 0.15276020765304565, 0.9289658069610596, 0.8224817514419556, -0.34406930208206177, 0.19672206044197083, -0.48859167098999023, -0.18369445204734802, -0.9120064377784729, 0.028488177806138992, 0.144779235124588, -1.9249638319015503, -0.7107139825820923, 1.332243800163269, 0.3704599142074585, -0.6394636034965515, -1.4019482135772705, 0.5645148158073425, 0.40701404213905334, -0.5297476053237915, 0.09533893316984177, 1.3380683660507202, -1.6590282917022705, -0.32666224241256714, -0.758118212223053, -0.04174191504716873, -1.2488648891448975, -1.1994190216064453, -0.773511528968811, 0.6823253631591797, -0.3874795436859131, -0.052486155182123184, -0.5023442506790161, -0.5130866169929504, -0.7735939025878906, 1.2256925106048584, 1.1993080377578735, 2.1111953258514404, 0.2797226905822754, 0.060717545449733734, 0.32018935680389404, 0.6789566874504089, -0.12899336218833923, -1.4185073375701904, 0.7009814977645874, -0.752640962600708, -0.975640058517456, 0.5575070977210999, -0.39833423495292664, 1.461382269859314, 0.46403926610946655, 0.46276187896728516, 1.4340267181396484, 0.7144790887832642, -1.2131240367889404, -1.8002341985702515, 0.6994110345840454, -0.35295379161834717, -0.5183676481246948, -0.7957909107208252], [-0.36101672053337097, -2.404808521270752, 0.02121567539870739, -0.29357030987739563, 0.6972755193710327, -1.0067499876022339, -1.4017465114593506, -0.40061259269714355, 0.9715200662612915, -0.5344617962837219, 0.5642703771591187, 1.5807033777236938, 0.6925009489059448, -0.943081259727478, -0.2759508490562439, -0.23222333192825317, -0.014378504827618599, -0.2922587990760803, -2.555047035217285, 0.5109870433807373, -1.400836706161499, 0.8026031255722046, -1.2541441917419434, -0.19932043552398682, 0.5590956211090088, 1.4807548522949219, 0.2612275779247284, 0.08521240204572678, 0.3176702558994293, -1.0175329446792603, -0.43776950240135193, 0.8085232377052307, 0.6348850727081299, -0.5907103419303894, -1.181140661239624, -0.7693197727203369, -0.917129397392273, 1.1994738578796387, 0.311284601688385, 2.2578189373016357, 0.8946730494499207, 0.4350021481513977, 1.0495080947875977, 0.8705738186836243, 1.2660590410232544, -0.21471837162971497, -1.0238101482391357, 0.4274204969406128, 0.051464978605508804, -0.29052650928497314, 0.7967418432235718, 1.7938740253448486, 0.6623108386993408, 0.5443024039268494, 0.12987682223320007, -0.49971142411231995, 1.321210503578186, 0.13648267090320587, -0.7309901714324951, 0.0023333861026912928, 1.5768083333969116, 0.3688845932483673, -0.20497570931911469, -0.5496179461479187], [1.0368194580078125, 0.42792537808418274, 0.09340299665927887, -0.7042489647865295, 0.10425468534231186, 0.5097887516021729, -0.2744397819042206, 0.7180579304695129, 0.6992683410644531, -1.9857816696166992, 1.4140342473983765, -1.0161200761795044, 0.13160055875778198, -1.061594009399414, -2.4547369480133057, 0.6341308355331421, -0.6592295169830322, 0.6281509399414062, -0.873860239982605, -0.1492534577846527, 1.74795663356781, 0.3535999059677124, -1.2502604722976685, 0.9077931046485901, -1.1833693981170654, -0.6885464787483215, -1.3651841878890991, 0.7906712293624878, -0.679135799407959, -0.4038558304309845, -0.8654788732528687, -1.9051384925842285, 0.8360745906829834, 0.9352989792823792, 0.1862390786409378, 0.4079729914665222, 1.5374099016189575, -0.7949997782707214, -0.7681766748428345, -0.2259558141231537, -0.12541598081588745, 1.180289387702942, -0.8673226833343506, -0.40705201029777527, -1.1390326023101807, 0.7509545087814331, 2.1299657821655273, 0.03046959824860096, -1.44932222366333, 1.7591081857681274, 1.149670958518982, 0.7978695631027222, -1.0122631788253784, 1.2337757349014282, 0.3233235776424408, -0.2584822177886963, -0.3247636556625366, 2.3169236183166504, -0.6009141802787781, 0.03226647898554802, -1.077439546585083, -1.0146335363388062, -0.8007129430770874, 0.5067716240882874], [-0.2540320158004761, -0.5107738971710205, -0.33891987800598145, 0.2969205677509308, -0.567510724067688, 0.5125366449356079, 1.4358748197555542, 0.28383544087409973, 0.3883918225765228, -0.9266955256462097, -0.8616152405738831, 0.765824019908905, 0.10517792403697968, 0.1272164136171341, 0.47338661551475525, -0.8596817255020142, 0.11246059834957123, -0.9084906578063965, 1.1578551530838013, -0.05884512513875961, 0.11806890368461609, 0.8199263215065002, -0.280163437128067, 1.8964080810546875, -0.42779117822647095, 0.33949917554855347, -0.2843874990940094, -0.1267504096031189, -0.2980055510997772, 1.5205955505371094, -0.8139581680297852, -0.49466609954833984, -0.9328704476356506, 0.29901161789894104, 1.7236279249191284, 0.06423439085483551, -0.35134217143058777, 1.7174283266067505, 1.9227888584136963, -0.7749377489089966, -0.7287750840187073, -1.7355362176895142, 0.656193196773529, 2.2750158309936523, -0.5489107966423035, 0.39874252676963806, 0.9186110496520996, -0.292520135641098, 1.5091356039047241, 0.30790552496910095, 1.6174858808517456, -1.728888988494873, -0.37292933464050293, 1.441009283065796, -0.08097710460424423, -0.49463415145874023, -1.6706771850585938, -1.1641185283660889, -0.679137647151947, 0.3502865135669708, -0.36736345291137695, 1.2835177183151245, -1.2679240703582764, -0.21474607288837433], [0.41022664308547974, -0.7286906242370605, 0.21146121621131897, -0.26452985405921936, -0.16976326704025269, -0.4889998435974121, -0.5438706278800964, -1.017903208732605, 1.1401796340942383, 0.1788109689950943, -0.9112181663513184, -0.3252708613872528, -0.38741806149482727, -0.11295055598020554, 0.7788584232330322, 2.417418956756592, 0.3607836961746216, 0.7801625728607178, 1.4554532766342163, 1.1305896043777466, 0.8225911259651184, -0.042120322585105896, -0.589830756187439, 2.1134188175201416, -0.019955145195126534, -0.5040011405944824, 1.6146695613861084, -1.104091763496399, -0.7637442946434021, 0.4320387542247772, 0.6008292436599731, -1.9740396738052368, 0.6895055770874023, 0.19485321640968323, -0.9982305765151978, -0.030567586421966553, 2.5498266220092773, 0.6909704208374023, -0.23447322845458984, -1.3913052082061768, 1.3844528198242188, 0.7291591167449951, 0.4652974307537079, 1.262664794921875, 0.8435492515563965, 0.8198339939117432, -1.1799885034561157, 0.8404229879379272, -0.32122939825057983, -2.3236782550811768, -1.965484619140625, -0.9096238017082214, 0.9735767245292664, 2.163379192352295, -1.1468044519424438, -1.776357650756836, 0.5756712555885315, -1.6277300119400024, 0.6561123132705688, -0.40336406230926514, 0.38239771127700806, 1.2617871761322021, -0.5580965876579285, 0.9896392226219177], [-1.0593854188919067, -0.68567955493927, -1.6376854181289673, -1.0052804946899414, 1.8267039060592651, 0.0452692024409771, -0.8976036310195923, 1.0611426830291748, 0.8189363479614258, 0.9555003046989441, 0.07153719663619995, -0.23373441398143768, 0.10738686472177505, 0.6091178059577942, -2.136347532272339, -1.161346435546875, 0.28325995802879333, 0.10535168647766113, -0.28215110301971436, 1.0611664056777954, 0.28867480158805847, -1.8224238157272339, 0.08356348425149918, 1.3227226734161377, 0.12014767527580261, 0.7110580801963806, 0.41336411237716675, 0.39399996399879456, -0.704393208026886, 1.9764906167984009, 0.2544139325618744, -0.9684781432151794, -0.012541188858449459, -1.0313889980316162, 0.9473108649253845, -0.8308702111244202, -0.3273015022277832, -0.813513457775116, -0.1962268352508545, -1.4531794786453247, -0.6582670211791992, -1.03826904296875, -0.48433414101600647, -1.2064859867095947, -1.7721781730651855, -0.6749807000160217, 0.8717024326324463, 0.26748576760292053, 2.2319345474243164, 0.3219696283340454, 0.803540825843811, 0.5937315821647644, -1.1376678943634033, -1.273202896118164, 0.7701507210731506, -0.2376057356595993, 1.3600797653198242, 0.6140503883361816, -0.33936047554016113, -0.5485855340957642, 0.44943174719810486, 1.1987755298614502, -0.020879041403532028, 1.7918307781219482]], [[-0.05032194405794144, 0.6139934659004211, -0.23868051171302795, 2.523383855819702, -1.9692003726959229, 0.1435449719429016, 0.2543344795703888, -1.1029877662658691, 1.2948834896087646, 0.46238139271736145, -0.4028625786304474, 1.1357922554016113, 1.7000876665115356, -0.005988330114632845, -1.1237859725952148, 0.2871800661087036, -0.48134729266166687, -0.820238471031189, 0.3639642596244812, -0.9464924931526184, -0.4401889443397522, -0.6861838102340698, -2.549450635910034, 0.5890542268753052, 0.46598368883132935, 1.4877076148986816, 0.8603625893592834, 0.5234678983688354, 1.411971926689148, -0.012995832599699497, -0.8506551384925842, -1.6067304611206055, 0.19141660630702972, -0.15698695182800293, -0.6407449841499329, 0.45397406816482544, 1.116400957107544, -1.557573914527893, -0.3569810390472412, -0.8119767308235168, -0.6583108305931091, 0.22017648816108704, 1.4277039766311646, -0.35621902346611023, 0.39081329107284546, 0.5853956937789917, -1.2213822603225708, 0.4037838578224182, -1.4302856922149658, 1.4405841827392578, 0.37119653820991516, 0.7672760486602783, 2.598022937774658, -0.12495653331279755, -1.185691237449646, -0.4659308195114136, 1.63821280002594, 0.6790518164634705, -0.05349921062588692, 2.857250452041626, 0.3786410689353943, 2.2057456970214844, 0.506767988204956, -1.090197205543518], [0.5623497366905212, 0.06257578730583191, -0.5678114295005798, -0.8482515215873718, -1.4899927377700806, -0.08928105235099792, -0.5565006136894226, -0.0492573156952858, 0.2656250596046448, 0.46544837951660156, 0.13529561460018158, 1.0828120708465576, -0.5152488946914673, 1.8166799545288086, 0.38586392998695374, 1.1942403316497803, -1.6697988510131836, 0.450723260641098, 2.236339569091797, 0.5284872651100159, -0.8699309229850769, -0.09814759343862534, -1.0799225568771362, -0.6702656149864197, -0.5960955619812012, -1.5042567253112793, 0.21825581789016724, -0.44191983342170715, 0.5672245621681213, 0.5713778734207153, 1.1344759464263916, 0.15094581246376038, 0.250661164522171, 0.04426807910203934, 2.3729453086853027, -1.0713257789611816, -1.6737059354782104, -0.5812190771102905, 0.3259201943874359, 0.06235616281628609, -0.3211696445941925, 0.6071818470954895, -1.1054290533065796, -1.470836877822876, -1.463175892829895, 1.5945584774017334, 0.14383834600448608, 0.10218512266874313, -1.3440488576889038, 0.8555316925048828, 1.0731728076934814, -0.16107764840126038, -0.6071434020996094, 1.4534235000610352, 1.0362776517868042, 1.784949779510498, 1.0070573091506958, 0.4021152853965759, 1.76524817943573, 0.2676483988761902, -0.5393209457397461, 1.6516741514205933, -0.16099779307842255, 0.8415421843528748], [1.5267927646636963, -0.6496158838272095, -0.4437173306941986, 0.5099347829818726, 0.5773559808731079, -0.9755963087081909, -0.3736696243286133, 0.09003733098506927, 0.35694459080696106, -0.26087242364883423, -0.3256400227546692, -1.4300899505615234, -1.4610087871551514, -0.995805025100708, -0.25363337993621826, -0.5662125945091248, 0.4670659303665161, -0.08204777538776398, 0.6808890700340271, -1.6838645935058594, -0.5669540166854858, -1.5096358060836792, 0.16963574290275574, 0.4220358431339264, 1.8801774978637695, 1.0459480285644531, 2.1228504180908203, -0.3594316244125366, -2.4395086765289307, 0.23665562272071838, -1.2758288383483887, 0.789622962474823, 1.3259435892105103, 0.20116764307022095, -0.522826611995697, 0.6133145689964294, -1.0780978202819824, -1.1930640935897827, -0.2000173032283783, 1.4341658353805542, 0.9574041962623596, 0.2512364089488983, 0.8930121064186096, -1.2718491554260254, 0.6822529435157776, -1.5237590074539185, -0.5666240453720093, 0.31001561880111694, 0.41453617811203003, 0.7000404596328735, 0.2930688261985779, -2.0182533264160156, 0.8933807015419006, -0.3056323826313019, -0.38434576988220215, 0.6710090637207031, -0.2196219265460968, -1.3272514343261719, 0.8395134210586548, -0.009404756128787994, -2.2670013904571533, 0.8185943365097046, -0.6723504066467285, 0.4836233854293823], [-1.5903394222259521, 1.7029894590377808, -0.9096994400024414, 0.08620722591876984, -1.1338791847229004, 3.0389111042022705, 1.1603294610977173, 0.665594220161438, 0.5402476191520691, -4.009869575500488, 0.7748368382453918, -0.9893496632575989, -0.3867564797401428, 0.161997452378273, 0.2628032863140106, 0.8509207367897034, -0.5988951921463013, -0.04408811032772064, -0.572694718837738, -0.39455440640449524, 1.165468454360962, 0.11579783260822296, 0.48275789618492126, -0.7565105557441711, 0.25840240716934204, 0.5366093516349792, 0.44966796040534973, 0.9166591763496399, 1.4412721395492554, 0.5466428399085999, -1.668561577796936, -0.05801486223936081, -0.6091936826705933, 0.38937222957611084, 0.7016289830207825, 1.3240957260131836, -0.6368206143379211, -0.7535066604614258, -1.0903046131134033, -0.7842862606048584, -0.8045253753662109, 0.9748955368995667, -1.410884141921997, -0.17973269522190094, -0.28792738914489746, -1.6575335264205933, -0.08199361711740494, -0.6646639108657837, -0.774258017539978, -0.5054036974906921, 0.6980558037757874, -0.45594754815101624, 0.2903386354446411, 0.4125041961669922, -0.2835600972175598, 0.38221392035484314, -0.10973434895277023, 1.0656815767288208, -0.2308972179889679, -0.08695570379495621, 0.323961079120636, 0.006597969215363264, -0.09539350122213364, 1.5765268802642822], [-0.1838994175195694, -1.1510480642318726, 0.8250682353973389, 1.052981972694397, -0.7767958045005798, -1.5440378189086914, -0.3313240110874176, 0.2883780598640442, 0.8313236832618713, -1.8371727466583252, 0.7593984007835388, 0.906252384185791, 1.6760927438735962, -0.36582037806510925, 0.028947288170456886, -1.0547879934310913, 2.4980080127716064, -0.3884139955043793, 0.25746950507164, 0.4405912756919861, -2.2834179401397705, -1.2506333589553833, 1.5536516904830933, -1.3697550296783447, -0.5615829825401306, -1.754393219947815, -1.810571551322937, 1.4253182411193848, 1.167500615119934, -0.8523988127708435, 0.3036983013153076, -0.3824409246444702, 1.200542688369751, -0.223196879029274, -0.5564466118812561, -1.3255301713943481, 0.01822555810213089, -0.8003758788108826, -0.18609589338302612, 0.5009996294975281, 1.0624680519104004, 0.400812566280365, -1.294636607170105, -2.2787208557128906, -1.18973970413208, 0.9201677441596985, 1.5378050804138184, -0.41924527287483215, -0.880224347114563, 1.2584599256515503, 0.15078113973140717, -0.33805903792381287, 0.40044882893562317, 1.340358853340149, -0.24918220937252045, -0.8095788955688477, 0.6758635640144348, -1.3020856380462646, -0.15856905281543732, 0.6022634506225586, 1.479243516921997, -1.0754783153533936, -0.8397020697593689, 0.7046645879745483], [-0.45450448989868164, -0.16039493680000305, -0.095054991543293, 0.9844350218772888, -1.9109941720962524, 0.5681086182594299, -0.6890646815299988, 0.10427597165107727, 0.2883285582065582, -1.3233520984649658, 0.17554804682731628, 0.5137053728103638, 1.5746444463729858, -0.4368470311164856, -0.24276532232761383, 0.05096418783068657, 1.6506460905075073, 1.630861520767212, -0.7386249303817749, -1.9955915212631226, -0.44527214765548706, 1.4117043018341064, -0.22905568778514862, -1.018355369567871, 0.5014851689338684, -0.7204133868217468, 0.5272627472877502, 1.4462329149246216, -0.4237446188926697, -0.1917094588279724, -1.0685911178588867, 0.060108501464128494, 1.7435303926467896, 1.1603891849517822, 0.7726275324821472, -0.29563459753990173, -0.7389194369316101, 0.02260305918753147, 0.21092116832733154, -0.956707775592804, -0.3788515627384186, 0.6698975563049316, 1.4148610830307007, -2.315169095993042, -0.5590934157371521, -0.06249712407588959, -0.5053567886352539, -0.9507376551628113, -0.604608952999115, 1.7173771858215332, 0.5456147193908691, -1.306547999382019, -0.19924409687519073, 1.987121820449829, 0.8079807758331299, -1.058716058731079, 0.0036232839338481426, -0.43971195816993713, 0.07832443714141846, -0.9640861749649048, 0.3849652409553528, 1.3412537574768066, -0.05717958137392998, 0.12563243508338928], [0.7549288272857666, -0.5388621687889099, -1.5879031419754028, -1.1830912828445435, 0.8412424325942993, 0.44811657071113586, 1.071799397468567, 0.6251472234725952, -0.0387142151594162, -0.3153130114078522, -0.34322455525398254, -0.23970189690589905, -1.1852130889892578, -1.5310955047607422, -1.3370285034179688, -0.2739446759223938, -0.8980713486671448, -0.5635215640068054, -0.5744501948356628, -0.7365420460700989, -1.3556358814239502, 0.8951015472412109, -1.6176578998565674, -0.3160950541496277, -0.6073072552680969, 0.3890746235847473, 0.9995114803314209, -0.9410274624824524, 0.8087924122810364, -1.0109103918075562, -0.8853089213371277, -0.4117686152458191, -1.204655408859253, -0.28257662057876587, 0.19086405634880066, 1.039374828338623, 0.9050760269165039, 1.1560308933258057, -1.1603890657424927, 0.46062400937080383, 0.3181462585926056, -0.4820282459259033, 2.662978410720825, -1.8642215728759766, 0.11057452857494354, -1.3162615299224854, -0.03879069164395332, 0.5129857063293457, 0.2447476089000702, 0.8669275045394897, 0.6228206753730774, 0.7997635006904602, -0.3042772114276886, -0.39318785071372986, -0.41228315234184265, -0.7945440411567688, -2.302886962890625, 0.5917810797691345, -0.3392362892627716, 1.7022323608398438, 0.06558477878570557, 0.20410284399986267, -1.4421257972717285, 0.15870238840579987], [0.06717565655708313, -0.7169013023376465, -0.39024195075035095, 0.12144628167152405, -1.6326804161071777, -2.173137664794922, 1.666619896888733, -0.21977204084396362, 0.06510767340660095, -0.9255166053771973, 0.9183664917945862, 1.6787360906600952, -1.0527347326278687, 1.0267071723937988, 1.626762866973877, -1.2699135541915894, -1.0191829204559326, 0.3764317035675049, 0.7844062447547913, -0.6883473992347717, 1.0226682424545288, 0.7633385062217712, -1.8494758605957031, -0.5998110175132751, -0.8096626996994019, 1.7812528610229492, -0.9320394992828369, 0.19294768571853638, -0.7071273922920227, 1.4492563009262085, -0.18748433887958527, -1.600466251373291, -0.1036151647567749, 1.1964755058288574, -0.564695417881012, 0.2945440411567688, -0.4563518166542053, -1.2841475009918213, 0.3288952112197876, 1.0710033178329468, 1.8880040645599365, 1.575494647026062, -0.9875901937484741, -0.48614397644996643, -0.43617287278175354, -0.1032378152012825, -0.049286242574453354, 0.7333593964576721, 0.9039545059204102, -0.7502629160881042, 1.075830340385437, 0.003872114699333906, -0.05528179928660393, 0.8517953157424927, -0.12566998600959778, 0.22714510560035706, 0.8963081240653992, 0.5301247835159302, -0.5135346055030823, -0.20050451159477234, -0.5200034379959106, 0.8367485404014587, 0.9041244387626648, -0.38343557715415955], [2.032392978668213, 0.6354747414588928, 1.2317049503326416, 0.9921294450759888, -0.36327236890792847, 1.3023813962936401, -0.508516252040863, -0.8277432322502136, 0.296862930059433, 0.2582179307937622, -0.9173373579978943, -0.089591845870018, 3.220625162124634, 0.14123617112636566, 0.8894318342208862, -0.8738709092140198, 0.11722008883953094, -0.14073322713375092, 0.602075457572937, -0.32451456785202026, -0.07649946212768555, -0.022134121507406235, 1.7436630725860596, 0.9705146551132202, -0.20746533572673798, 0.6989296078681946, -0.4262044131755829, -1.242477536201477, 0.42945703864097595, 0.18536600470542908, -0.18071700632572174, -0.8416637182235718, -1.1728541851043701, 0.1939162313938141, 0.46102362871170044, 0.3610829710960388, 0.39834171533584595, -0.3719608187675476, -0.344663143157959, -0.716894805431366, -0.6788580417633057, -1.338962197303772, -0.502403736114502, 0.4895433783531189, -1.656043529510498, 0.5361753106117249, 0.13550472259521484, -0.4355325996875763, -1.9370359182357788, -0.6774973273277283, 0.9586169719696045, -1.1814243793487549, -0.5001108646392822, -1.3301135301589966, 0.6942260265350342, -0.031177416443824768, -0.2853785753250122, 0.40390992164611816, 0.03313208371400833, 1.361219048500061, -0.31205639243125916, 0.20567163825035095, -0.09312168508768082, -2.306030035018921], [0.03652653843164444, -0.37738025188446045, -0.7891831398010254, 0.23848196864128113, -0.08184440433979034, -1.3187048435211182, -0.26657766103744507, -0.49549704790115356, -0.0025524974334985018, 0.7547307014465332, -0.24657954275608063, 0.016358118504285812, 0.7469179630279541, -1.3660368919372559, -0.8576256632804871, -1.982079267501831, 2.079982042312622, -1.2094379663467407, -0.2924533486366272, 0.43223294615745544, -0.3735100030899048, -0.8076034188270569, -1.1471400260925293, -0.41143471002578735, -0.00553514901548624, -1.2943562269210815, -0.6760528683662415, 2.0691981315612793, 0.2447412759065628, 2.041032314300537, 0.8867286443710327, -1.0852025747299194, 1.3607375621795654, -0.11382018029689789, 0.23732879757881165, 0.027693066745996475, -1.4662317037582397, -1.5939682722091675, 1.0332859754562378, 0.15544386208057404, -0.22679798305034637, 1.0980712175369263, -0.65547776222229, 0.6435108184814453, 2.317047357559204, -1.9848805665969849, 2.016146183013916, -0.3077251613140106, -0.47822996973991394, 1.2623533010482788, -0.37924525141716003, 0.40325963497161865, -0.5342373251914978, 0.20040598511695862, 2.4253785610198975, -0.18493451178073883, -1.4119046926498413, -0.46800416707992554, 0.1273290514945984, -0.7988114356994629, -0.07848253101110458, -1.6837903261184692, -1.286149024963379, -1.3330953121185303], [0.05122704803943634, 0.2786497473716736, 0.2931724190711975, -0.22711458802223206, 0.5333088040351868, -0.9660214781761169, -0.9857478737831116, -0.7315292954444885, 0.3481466472148895, -0.1498592346906662, 1.7701586484909058, -0.4046269655227661, 0.2808101773262024, -0.43796178698539734, -0.5025380849838257, -1.0463932752609253, 0.10963068157434464, 0.11644898355007172, -0.07677394896745682, 0.2523408532142639, -0.10422442108392715, -0.4415571689605713, 0.901329517364502, 1.302785873413086, -1.4367151260375977, -0.21727201342582703, -0.16252680122852325, 0.4872787892818451, 0.5578277111053467, 0.5915714502334595, 1.5488187074661255, -0.1180860698223114, 0.6009745001792908, 0.12091991305351257, -2.4298436641693115, -0.8018500208854675, 1.5306570529937744, -0.0833669900894165, 1.3190206289291382, -0.4429263770580292, 0.874699592590332, -0.15870891511440277, 0.8372701406478882, -0.5976212620735168, 0.5323631167411804, -1.1740739345550537, 0.21339361369609833, 0.404167115688324, -1.137555480003357, 1.7118103504180908, -0.554524302482605, 0.09565021097660065, 1.992106318473816, -0.764118492603302, 0.23841580748558044, -0.7282887101173401, 0.19394633173942566, -0.07182197272777557, -0.3476918339729309, -1.9167569875717163, -0.03673470765352249, -0.5485442280769348, 0.3229328691959381, -0.54071044921875], [-0.6599725484848022, -0.3037644326686859, -0.3467783033847809, -1.167272686958313, 1.9013456106185913, 0.6527412533760071, -1.8585273027420044, -1.5771644115447998, 1.03816819190979, 0.588490903377533, -0.006065999157726765, 0.4751916527748108, 1.5200543403625488, -0.3576442003250122, 2.091392755508423, 1.458184003829956, 1.3121552467346191, 0.3449932634830475, 0.022438276559114456, 0.10596717149019241, 0.26848849654197693, 0.5693750381469727, -0.36579039692878723, -0.5668138265609741, 1.8888288736343384, 1.1766501665115356, 0.5238719582557678, -0.5643908977508545, 0.8030316233634949, -0.7135840654373169, 0.748611569404602, -0.10824970155954361, -1.0444350242614746, 0.37783777713775635, -0.745919942855835, -1.1951326131820679, 1.468966007232666, 0.7626032829284668, 0.7249035835266113, -0.36458200216293335, 1.516045331954956, 0.5747963786125183, -0.629179060459137, 0.9130244255065918, -1.1677227020263672, -0.14339075982570648, 1.2139770984649658, -0.6950072050094604, -1.9501962661743164, 0.542930006980896, -0.7917932271957397, -1.349891185760498, -0.2840479016304016, 1.0352295637130737, 0.6735486388206482, 1.2557995319366455, -0.5235954523086548, -0.8543421626091003, -0.6671690344810486, 1.1196507215499878, -1.2905746698379517, 0.9422374367713928, 1.0978070497512817, -0.9713641405105591], [-0.5139659643173218, 0.6906929612159729, 1.2386126518249512, -0.03710222616791725, -0.3000011146068573, -0.052269939333200455, 1.953505277633667, 0.4590606689453125, 0.9812326431274414, 0.7503272891044617, 0.016946088522672653, -0.9597927331924438, -0.10020221024751663, -0.5706336498260498, -0.5797598361968994, 0.7848669290542603, -0.10091566294431686, -2.251840829849243, -0.621372640132904, 0.46075019240379333, 1.7284152507781982, -0.15609878301620483, -0.873856782913208, -1.4743421077728271, -0.7659716606140137, -0.20279447734355927, 1.1535253524780273, -0.36666861176490784, 1.2553989887237549, 1.4333748817443848, -1.5310277938842773, 1.0287299156188965, 1.3671743869781494, 0.2157285511493683, 0.468569278717041, -0.8929482102394104, 0.2575056850910187, -0.06658212840557098, 0.5069475173950195, 1.0169097185134888, 1.7020225524902344, 0.48130643367767334, 0.0044777896255254745, 1.8328782320022583, 1.8200567960739136, 0.6633699536323547, 1.261907696723938, 0.29469746351242065, -0.7019959688186646, 1.1237716674804688, 0.12959565222263336, -1.1617926359176636, -1.5800230503082275, -0.41964924335479736, 1.8738189935684204, -0.7749502658843994, -0.327880859375, -2.8690097332000732, -0.16402190923690796, 0.11378935724496841, -0.7964664697647095, -0.22323957085609436, -0.06471472978591919, 0.4508940279483795], [0.6107978224754333, -1.811559796333313, 1.285079836845398, -0.18175257742404938, 2.7832350730895996, 1.7132346630096436, -0.31668540835380554, -0.505645751953125, -1.4550747871398926, 1.543052077293396, -1.8114778995513916, 0.04378893971443176, 0.7756555080413818, 0.13698671758174896, -0.4364147186279297, -0.1664378046989441, -0.7591180205345154, -0.4139571785926819, 0.6071743369102478, 0.6739818453788757, 0.48674896359443665, 0.48240429162979126, 0.47209325432777405, 0.30740708112716675, 0.30380213260650635, 0.44008398056030273, 0.6751828789710999, -0.3980139195919037, -1.017146110534668, -1.2872099876403809, 1.3582149744033813, -0.5559400916099548, -1.3532651662826538, -1.2807759046554565, 0.10172668844461441, 0.4462323486804962, -0.4863033890724182, -0.6627179980278015, -0.26527804136276245, 0.09781096130609512, 1.1214088201522827, 0.8693507313728333, -1.1590282917022705, -1.3267570734024048, -1.3207595348358154, 0.7432542443275452, -1.221901535987854, 0.9352620840072632, 0.10350649803876877, -0.40715664625167847, 0.49179714918136597, -0.40440401434898376, -0.662121593952179, -2.181912422180176, 0.0472152978181839, -0.6012957096099854, 0.5028010606765747, -0.7289372086524963, -0.49926483631134033, -1.0161786079406738, 0.7443501353263855, 0.7230129837989807, 1.2880936861038208, 1.6476202011108398], [2.2729945182800293, -0.006777357310056686, -0.5689036250114441, -2.041499137878418, -1.0118361711502075, -0.6340281963348389, 1.4412343502044678, -2.5068936347961426, 0.375969260931015, 0.7214213013648987, -0.08269581198692322, -0.3671095371246338, -0.31172096729278564, 1.0849924087524414, -0.39355573058128357, 0.5447525382041931, 0.5029905438423157, -1.5230306386947632, 1.3951094150543213, 0.5965358018875122, -0.02539958246052265, 0.40477317571640015, 1.2272354364395142, -1.1143825054168701, 0.07978679984807968, 1.1058971881866455, 0.5153540968894958, -0.24940161406993866, -0.6101164817810059, 0.4258357584476471, 0.2344687283039093, -0.6730928421020508, -0.9005348682403564, -0.5899239182472229, -1.5630204677581787, -0.43101730942726135, -0.1966298222541809, 0.5787699818611145, -0.2556804120540619, 0.8393411636352539, -0.5717569589614868, -2.8085415363311768, 0.48209047317504883, -0.7922437191009521, -0.2809033989906311, -1.4455265998840332, 1.4298069477081299, 1.954393744468689, 0.9512761235237122, -1.2506365776062012, -2.2817153930664062, -0.808798611164093, -0.6503036022186279, 0.31462711095809937, 0.8905078768730164, 1.6470675468444824, -1.2641373872756958, 0.6185299754142761, 0.6127364635467529, -0.003708516014739871, -1.70572829246521, -0.6811551451683044, 1.1491526365280151, -0.9639497995376587], [0.6138075590133667, -0.7804048657417297, -0.5515170693397522, 0.5952267646789551, 2.082597017288208, 1.614495873451233, -0.13497214019298553, 0.651212215423584, 0.7504191994667053, 0.4524950087070465, 0.03291953355073929, 0.7788712382316589, 1.0630205869674683, 0.5939504504203796, 0.33959442377090454, -0.1677936166524887, -0.6985142230987549, -0.8203595280647278, 0.3564254939556122, 1.5751147270202637, -1.187698483467102, -1.364661455154419, 0.796835720539093, 0.4550289511680603, -1.9260238409042358, -0.6704905033111572, 0.5028225779533386, -1.1879826784133911, 1.3077137470245361, 0.11855137348175049, -0.24952280521392822, 1.0651865005493164, -0.11892619729042053, 0.0195216815918684, -0.15793731808662415, -0.2844080924987793, -0.8988754153251648, 1.604205846786499, -1.361932396888733, 0.4798542857170105, -0.8209543824195862, 0.24756698310375214, -0.8958733677864075, 0.24139179289340973, 0.9636626243591309, 0.060161322355270386, -0.044173434376716614, 0.07252555340528488, 0.9092724919319153, -0.9919822216033936, 0.061316922307014465, 1.1916968822479248, 0.7349315881729126, 0.5645561814308167, -0.3507443070411682, -0.9651309251785278, 1.5714330673217773, -0.07539904862642288, 0.0018416730454191566, 0.7765771150588989, 0.1893942952156067, -0.7300037145614624, 0.9438152313232422, -0.5033249258995056]], [[-0.006436409428715706, -0.19210349023342133, -0.8998745083808899, 0.5802339315414429, -1.351345419883728, 0.9874894618988037, 0.2844283878803253, -0.6952989101409912, 0.47960996627807617, -1.4776638746261597, 3.0881550312042236, -0.09718068689107895, -0.3245948851108551, -0.766045093536377, 0.8254354000091553, -0.27443382143974304, 1.20597505569458, 0.7353302240371704, 0.1459672749042511, -1.2241711616516113, -0.7552110552787781, -0.8402437567710876, 1.5246535539627075, 0.14366571605205536, 0.690287709236145, 0.6545976996421814, -1.2885453701019287, -1.5189073085784912, -0.6077395081520081, -2.701706886291504, -0.07415740191936493, -0.23164938390254974, -0.3874525725841522, 1.147491693496704, 0.7093045711517334, -0.21435318887233734, -0.39519333839416504, 0.19055485725402832, -0.1735822558403015, 0.522389829158783, 1.508697509765625, -0.12400813400745392, 1.2801620960235596, 0.7442874312400818, -0.031353216618299484, -0.20425771176815033, -0.8669273257255554, 0.531011700630188, -0.5894597768783569, -0.04701739177107811, 1.523608684539795, 1.3168299198150635, -0.19977059960365295, -0.599025547504425, 1.2069859504699707, -0.2901035249233246, 0.5785166025161743, -1.8968044519424438, 1.9741309881210327, -1.3445223569869995, 0.7773278951644897, 1.4107508659362793, -0.18872417509555817, 1.0036765336990356], [0.949551522731781, 0.36149320006370544, -0.4757373631000519, 0.02216433547437191, -1.5332553386688232, 0.6725612282752991, 1.0991212129592896, 1.7869211435317993, 0.6910924911499023, 0.08349140733480453, 0.743205189704895, -0.5366328954696655, -1.6528481245040894, -0.2322310358285904, 0.07270140200853348, 0.29145461320877075, 1.2608191967010498, 0.8187901973724365, 1.0071907043457031, 1.1828370094299316, -1.9676803350448608, 1.4319236278533936, 1.835747480392456, 0.055624570697546005, -0.18363741040229797, 0.5396456122398376, -0.009837500751018524, -0.5472429394721985, 0.39753827452659607, -0.9621347188949585, 1.387154221534729, -0.743014931678772, -0.8185508251190186, -0.4043969213962555, 1.699175238609314, -1.2105787992477417, 0.34095317125320435, -0.7850145101547241, -1.3987778425216675, 1.5268300771713257, 0.21507145464420319, 1.2769237756729126, 1.1635388135910034, -0.5320321917533875, -0.48314228653907776, -0.0567755401134491, -0.15821124613285065, 0.4968114197254181, 1.5424776077270508, -0.7670826315879822, -0.514076828956604, 1.2646218538284302, 1.4891350269317627, -0.3359571695327759, 0.3553617298603058, -0.2614453434944153, 1.641715168952942, 0.6055487394332886, 0.23574253916740417, 0.9461084604263306, 0.8661386966705322, 0.014353002421557903, 0.7728404998779297, 0.49074217677116394], [0.3015778660774231, 0.07186924666166306, 0.9797871708869934, 1.5035611391067505, -1.2079060077667236, 0.28200653195381165, -1.273789644241333, -0.49447232484817505, -0.8022061586380005, -1.1422147750854492, 0.3845566213130951, -0.07880831509828568, -0.2797462046146393, -0.2117997109889984, -0.7794296145439148, -0.25885093212127686, -0.6857959032058716, -0.1840469241142273, -0.5255358815193176, 0.09622218459844589, 0.21946051716804504, -0.14532431960105896, 1.2611384391784668, 0.282662957906723, -0.2267674058675766, 0.5794154405593872, -2.506547689437866, -1.1932603120803833, -0.28661951422691345, -0.37268608808517456, -0.7509964108467102, -0.47422224283218384, 0.4342886209487915, -0.580693244934082, 0.5164502859115601, 0.7713235020637512, -0.056726183742284775, 0.8488770127296448, -0.6729886531829834, 1.0417062044143677, 0.9070141315460205, 0.2591980993747711, 0.3750735819339752, -1.817098617553711, -1.0609043836593628, -0.8446151614189148, -0.023117007687687874, 0.41116437315940857, -0.23650896549224854, 0.6399446725845337, 0.579979419708252, -2.745056390762329, 0.2663349509239197, -0.1778038591146469, -0.900150716304779, -0.3102705776691437, 0.5989768505096436, 0.6780032515525818, -0.1633123904466629, -0.372787743806839, 0.4830649793148041, -0.7721054553985596, 1.585675835609436, -0.47952598333358765], [-0.03826751932501793, -1.62811279296875, -0.019252339377999306, 0.26920974254608154, -1.615921139717102, -0.4785243272781372, 0.6743799448013306, 1.2658787965774536, 0.6977770328521729, 0.2395053505897522, 0.0255263838917017, 0.8157777786254883, 0.20511503517627716, -0.5004540085792542, -1.8715367317199707, -0.938057005405426, 0.1222919225692749, 0.5792638659477234, -0.44415587186813354, -0.09548776596784592, 0.09961912781000137, -0.2438475340604782, -1.0604742765426636, -0.37043577432632446, 0.5151888728141785, 0.8797526955604553, -0.5194955468177795, -2.295898675918579, -1.4077913761138916, -2.059330701828003, -1.6580246686935425, 0.3398677408695221, -0.535933792591095, 1.3181771039962769, -1.0468428134918213, 0.03360067680478096, 0.14650218188762665, 0.3161962628364563, -1.7830109596252441, -0.38035619258880615, 0.9328957200050354, -1.824619174003601, 2.4176032543182373, 0.05620897561311722, 0.32961705327033997, 1.295322299003601, 0.076776884496212, 0.9897280335426331, -2.3055200576782227, -0.2163306623697281, -1.2842813730239868, 0.8382542133331299, -0.18160472810268402, -0.6860418915748596, 0.7826744318008423, -2.131570339202881, 1.0977790355682373, -0.8405455350875854, -0.6743128895759583, -2.415963888168335, 1.0965465307235718, 1.165540099143982, -0.2364213615655899, -0.48562493920326233], [0.13329526782035828, 1.8002870082855225, -0.6350920796394348, -0.5395434498786926, -2.5219082832336426, -2.1231887340545654, -0.3124339282512665, -0.5729196071624756, 0.8533511757850647, -0.5683585405349731, 0.4552546739578247, 0.49115702509880066, 1.9040759801864624, -0.44925013184547424, 0.3635425269603729, 1.065608024597168, -1.5035876035690308, 0.42536547780036926, -1.2618591785430908, -0.5671479105949402, -1.3753377199172974, -0.35356128215789795, 0.22020097076892853, 1.568140983581543, -0.2120458036661148, 0.04965806007385254, -0.7376015186309814, -0.11092355847358704, 2.0685744285583496, -0.14693346619606018, -0.0642213523387909, -0.8882770538330078, 0.5668491721153259, 0.816791832447052, 1.15696120262146, -0.47981634736061096, -0.6745662689208984, 1.2690201997756958, -0.025837915018200874, -0.027157265692949295, 0.7223012447357178, 0.22398890554904938, -1.1308881044387817, 0.922382116317749, 0.21909092366695404, -2.0574467182159424, -0.7202380299568176, -0.4272400140762329, -1.6312682628631592, -0.6428537368774414, -0.03276880085468292, 0.3281231224536896, 0.059970077127218246, -1.4508767127990723, 2.6595821380615234, 0.6596378087997437, 0.5055230259895325, 0.5309200882911682, -0.521687924861908, -1.4076288938522339, -0.19864284992218018, 0.20106665790081024, 0.1485731601715088, 0.5604261159896851], [1.090460181236267, -1.6225873231887817, -1.185227394104004, -1.2123740911483765, -0.14287538826465607, -0.41884875297546387, 0.06832647323608398, -0.4793623685836792, 0.42367202043533325, -0.3771705627441406, 0.5358504056930542, -1.0751312971115112, -0.15462976694107056, -0.9408121109008789, 0.5517317652702332, 0.23616020381450653, -0.293455570936203, -0.2943045198917389, 0.019552353769540787, 0.2648095488548279, -1.424153208732605, -2.477937936782837, -0.05648721009492874, 0.7093430161476135, 0.26831674575805664, -0.8508980870246887, 1.1323968172073364, -1.204554557800293, -1.5385011434555054, -0.9420133829116821, 0.542188286781311, 0.9586615562438965, -0.4668351411819458, -1.0935144424438477, -0.5600430369377136, 0.5475242137908936, 0.3658609986305237, 0.43610307574272156, 0.9139801263809204, 1.4618197679519653, 0.08562278002500534, -0.0794532299041748, -0.9460894465446472, -0.47479650378227234, -1.928501009941101, 0.03534305468201637, -1.7782341241836548, -1.2334063053131104, 0.7086160778999329, 0.16141599416732788, -0.3502286374568939, 0.18193255364894867, 0.7829709053039551, -0.6715757250785828, -1.5602309703826904, 3.1302731037139893, 1.1999797821044922, 0.23170535266399384, -1.0873098373413086, 0.9609846472740173, -0.6542973518371582, -1.119627594947815, -0.08508910983800888, -0.38308146595954895], [-0.2305998057126999, 0.3165000379085541, 0.7937287092208862, -0.44971442222595215, -0.9482747912406921, 0.37491917610168457, 1.401442050933838, 0.6051344275474548, 0.5308927297592163, -2.424321413040161, 0.8395231366157532, 0.770418107509613, -1.8848494291305542, 0.20809419453144073, 2.1188535690307617, 0.8030587434768677, -1.226096510887146, 1.1239272356033325, -0.6133702993392944, -1.0142648220062256, 0.01637391746044159, -1.2323949337005615, -0.975182056427002, 2.2155046463012695, -2.0845606327056885, 1.2404117584228516, 1.4000617265701294, -0.37422120571136475, 0.7019509673118591, -0.05947740748524666, -0.1372123807668686, 0.010648838244378567, 1.5789762735366821, 1.415154218673706, -0.034286078065633774, -0.7015349864959717, -0.7448868155479431, -0.6894184350967407, 1.5360785722732544, 0.39733636379241943, 1.6213339567184448, 0.6437059640884399, 1.279622197151184, -1.6277321577072144, 1.1801104545593262, -1.4723716974258423, 0.8876132965087891, 1.2666562795639038, -1.1273006200790405, 0.701999306678772, -0.10820434987545013, 0.08032199740409851, 1.393279790878296, -0.30645951628685, -1.2733657360076904, 1.3296735286712646, 0.8834463953971863, 0.24200454354286194, -1.9533261060714722, -0.7905541658401489, 0.4576599895954132, -1.7353435754776, 1.6527961492538452, 0.5227649807929993], [-0.8548425436019897, -1.1645249128341675, -0.17947348952293396, 1.0480997562408447, -2.2876763343811035, -0.433449387550354, -0.0950261577963829, 0.7321255207061768, -0.7719886898994446, 0.9442986249923706, -0.018111957237124443, -0.7550591230392456, 1.6773898601531982, -0.1680290549993515, -0.6755351424217224, -0.01133478432893753, 0.6211205124855042, 0.13898234069347382, 0.4643740653991699, 0.7891255021095276, -0.5476727485656738, -1.8372292518615723, -0.052075568586587906, -1.679836392402649, 0.12236287444829941, -0.19870200753211975, -1.520155668258667, 0.07702682167291641, 0.8067070841789246, 1.0258796215057373, 1.871594786643982, -1.084438681602478, 0.8292240500450134, 0.2339237928390503, -0.28753623366355896, 0.8050916790962219, -0.5244303941726685, 1.1573668718338013, -1.711353063583374, 0.05860935524106026, -1.199881672859192, -0.05604347959160805, -0.48670798540115356, 0.1977744996547699, -1.7229893207550049, 0.7120625972747803, -1.8053700923919678, 0.708832323551178, 0.661520779132843, 0.03671848401427269, 2.0957815647125244, -0.5563021302223206, 1.5333099365234375, 0.16184620559215546, -0.6303080320358276, 1.3821340799331665, -0.5697161555290222, -0.21212804317474365, -0.22861360013484955, 0.20780837535858154, -0.3026712238788605, -0.5049097537994385, 0.15344573557376862, -0.2730654776096344], [-0.681043803691864, -0.7394797801971436, 0.8580759763717651, 0.3595867455005646, -0.5463042259216309, 0.7535814642906189, 0.07399840652942657, 1.1669539213180542, -0.6791395545005798, 0.4252035319805145, -0.4332229495048523, -0.7045774459838867, -2.104712724685669, -0.4828242063522339, 0.4678173363208771, -0.7499746084213257, -0.11358998715877533, 1.134636402130127, -1.4090512990951538, 0.6484652757644653, -1.9863054752349854, 0.7081549167633057, 0.5417935252189636, 1.7500865459442139, -0.25813916325569153, 0.3130358159542084, -1.8116217851638794, 0.9230989217758179, -1.09224534034729, -2.4940474033355713, 0.14580382406711578, 0.7452004551887512, -1.2160990238189697, -0.2841338515281677, -0.14315176010131836, 0.8995679616928101, 0.5438368916511536, -0.6116459965705872, 0.6913975477218628, 0.6102914214134216, 1.380023717880249, 0.8317825198173523, 0.844399094581604, -0.5000507831573486, -0.3054315447807312, 1.4194566011428833, -0.7809796333312988, -0.03609707951545715, 1.148540735244751, -0.7827228903770447, 0.49925416707992554, -0.00792770180851221, 2.486891031265259, 2.083360433578491, 1.066318154335022, -0.7445684671401978, 1.128781795501709, -1.3605412244796753, 0.9026252627372742, -0.4592742919921875, -1.5610450506210327, 0.4416603446006775, 0.27717286348342896, 1.0924649238586426], [0.4453706443309784, 0.7495551705360413, -0.37009501457214355, -1.2766592502593994, -0.3654617369174957, -0.32780227065086365, -0.11832985281944275, 1.545218586921692, -0.43471601605415344, 0.1806495636701584, -0.21142084896564484, 0.6488543152809143, -1.2958629131317139, 0.12964583933353424, 1.0713061094284058, 0.5835492610931396, 0.8730122447013855, -0.11535753309726715, -0.5229129195213318, -1.2422469854354858, 1.198825716972351, -2.14821457862854, 0.33142945170402527, -0.7513898015022278, 1.1046892404556274, 0.11005441844463348, 0.11035091429948807, -1.1970651149749756, 1.748967170715332, 0.9211442470550537, 2.4682059288024902, 1.4823416471481323, -0.14949621260166168, 1.2571223974227905, 0.3519109785556793, 0.5845890045166016, -0.17625723779201508, -0.3021785616874695, 1.0585111379623413, 1.3177180290222168, -1.418432593345642, -0.01606173999607563, -1.1966780424118042, -0.022009264677762985, 0.2771218419075012, -1.034251093864441, 1.7580187320709229, 1.5240283012390137, 0.34167835116386414, -0.10496275871992111, 0.768083393573761, -0.036554962396621704, -0.5587934851646423, -0.49697786569595337, 1.4584616422653198, -0.07241977751255035, -0.45716455578804016, -0.5440634489059448, 0.8486869931221008, 0.3137184679508209, 0.9482271671295166, 0.6542477011680603, 0.9472557306289673, -0.002082705730572343], [-2.1762893199920654, 0.8430951237678528, -0.5278239846229553, -0.5037903189659119, -0.7067850828170776, 0.832605242729187, -0.5156419277191162, -0.25618335604667664, -2.155061721801758, -0.40895527601242065, 0.9542035460472107, -0.19625233113765717, 0.39355164766311646, 0.9318352937698364, 1.2681671380996704, -0.020871581509709358, 1.2236398458480835, 0.20851777493953705, -0.21582375466823578, 0.9442683458328247, -1.7584612369537354, 0.697783887386322, 0.7590191960334778, -0.9836622476577759, 1.1162962913513184, 0.09802883118391037, 0.1324395090341568, -1.6504253149032593, 0.21562252938747406, 0.19245220720767975, -0.641697347164154, -0.3091772794723511, 0.3870202898979187, -0.6215901374816895, 0.23426738381385803, 0.6473957300186157, 0.19798831641674042, -0.14414553344249725, -0.1719684600830078, -2.0268261432647705, -0.8903615474700928, 0.7742050290107727, -0.011490018106997013, -0.8855294585227966, 0.1899072378873825, 1.0684101581573486, 0.06763599067926407, -0.8227362632751465, -0.43021780252456665, -1.0448187589645386, 1.1578365564346313, 0.8564876914024353, -0.5444551706314087, 0.5349248647689819, -0.9545013308525085, 0.07158441096544266, -2.0629913806915283, -0.44432005286216736, 1.756797432899475, 1.2428268194198608, -0.1959114372730255, 0.4423297345638275, -0.8889158964157104, 0.3318384885787964], [0.03841913491487503, 0.2599940896034241, -0.08467300981283188, -1.4068386554718018, -0.909942090511322, -0.25046834349632263, 0.9637173414230347, -2.499584913253784, -1.0374373197555542, -0.919711709022522, -0.2388239949941635, 0.05966068431735039, -2.157559633255005, 0.882551372051239, 1.3213558197021484, 1.547763466835022, 1.4913614988327026, 1.8407312631607056, -0.48469069600105286, 0.23360490798950195, -0.2500046193599701, 0.021403564140200615, 0.8932451605796814, 0.11657141149044037, 1.594622254371643, 2.0259149074554443, -2.510436773300171, 1.0999785661697388, -0.09135712683200836, 2.0110254287719727, 1.083349347114563, -0.5102695822715759, -1.172066330909729, 0.658111572265625, 0.5630711317062378, 1.5032364130020142, 0.5958184599876404, -0.8393847346305847, 0.1517830640077591, 1.4063489437103271, -0.32552722096443176, -0.8105606436729431, 1.6118282079696655, -0.4306069612503052, 0.5698582530021667, 0.319499671459198, 2.391166925430298, 1.939727783203125, 2.1000423431396484, -1.089426875114441, 0.029925765469670296, -3.237097978591919, -1.2415090799331665, -1.035822868347168, 0.35897842049598694, -0.6894528865814209, -1.8447669744491577, 1.3842267990112305, 0.05331944301724434, -0.3838063180446625, -1.3463939428329468, 0.12829647958278656, -0.06354330480098724, 0.9389973878860474], [-1.3487437963485718, -0.20119349658489227, -0.5453379154205322, -1.3120907545089722, -0.06286722421646118, -0.28020310401916504, -1.661022424697876, 0.6705498695373535, -0.11133460700511932, -0.5132566690444946, -1.0536648035049438, 1.2347701787948608, -0.1316789984703064, 0.5732832551002502, 0.8519999980926514, -1.0469751358032227, 0.5348936319351196, -0.3932737708091736, -0.308383971452713, -1.8786742687225342, 0.13592813909053802, 0.23692595958709717, -0.7138814330101013, -1.2337130308151245, -1.2072854042053223, 0.1382429301738739, 0.4027647078037262, -0.48216959834098816, -1.1589651107788086, 0.41897305846214294, 0.3565202057361603, 0.7240332961082458, 0.23371078073978424, 0.44661104679107666, 0.4232490658760071, -2.3353428840637207, -1.4565515518188477, 1.4148139953613281, 0.3213365375995636, 1.5128859281539917, 0.7221654057502747, 0.1881287842988968, -0.2084309458732605, 0.5086467862129211, 1.2553120851516724, 0.03273915499448776, 0.4585888385772705, -0.4154229164123535, -0.2843814790248871, 0.10819333046674728, 0.5538269281387329, 0.8146688938140869, 0.5393084287643433, -0.7363662123680115, 0.0014896474312990904, -0.2859335243701935, 0.7677155137062073, 1.2323129177093506, 1.731413722038269, 0.3587796986103058, 0.4814854562282562, 0.11059239506721497, 1.0226233005523682, -0.6950884461402893], [0.5710785984992981, -0.28966471552848816, -0.9087542295455933, 1.354811429977417, -0.6004570722579956, -0.3725983500480652, -0.3823675215244293, 0.5808518528938293, 0.0542878732085228, 0.5156216025352478, -0.12940675020217896, 1.1257774829864502, -0.7015636563301086, 0.9896633625030518, -0.03017430752515793, 1.2266809940338135, -0.057797010987997055, -1.5641816854476929, -1.4538809061050415, -0.15181460976600647, 0.00856236182153225, -0.5070018768310547, -0.5758419036865234, 0.051470428705215454, -0.7030662894248962, -1.461174488067627, 0.4768659472465515, -0.6257140040397644, -0.2787543833255768, -2.3807389736175537, -0.3405193090438843, 0.1138676255941391, 0.7679755091667175, 0.3494984209537506, 0.09853767603635788, 1.107571005821228, -1.252393364906311, 1.3363733291625977, 0.6877326965332031, -0.8276675939559937, 0.7465289831161499, 0.7015155553817749, -0.25680118799209595, -1.7247508764266968, -1.4337880611419678, -0.5965878367424011, -0.04603644460439682, -0.26955074071884155, 1.3205194473266602, -0.9591459631919861, -0.07210661470890045, -1.4378461837768555, -0.5291209816932678, 0.21893540024757385, -0.5559883117675781, 1.250417709350586, 0.19884736835956573, 1.1518583297729492, 0.8827476501464844, 0.37402522563934326, 0.014726463705301285, 0.2600557506084442, -0.024775667116045952, -1.107936978340149], [-0.3437410295009613, 0.02297825925052166, 0.5949034094810486, 0.3685712218284607, 1.1808360815048218, -1.2164233922958374, 2.0662827491760254, -0.6945061087608337, 0.028360804542899132, -0.33831945061683655, -0.8323983550071716, -0.9560925364494324, -0.044615983963012695, 0.9452469348907471, -0.410899817943573, -1.5000317096710205, 1.620188593864441, 0.7094306945800781, -0.12925086915493011, 1.6813782453536987, 0.6914003491401672, 0.3524070680141449, 3.1934125423431396, -0.6605128645896912, 0.6267614364624023, 0.23094087839126587, -0.7421813607215881, -1.3822264671325684, -0.8329029679298401, 0.23070353269577026, 1.6225011348724365, -0.713153600692749, 0.8700149059295654, -0.2100970447063446, -0.18142203986644745, -0.6496198177337646, -1.5872172117233276, -1.0488898754119873, -0.9381593465805054, -2.842759847640991, 0.8368681073188782, 0.028677061200141907, -0.18406642973423004, 1.8338898420333862, -2.1780264377593994, 0.11727269738912582, 0.8581454753875732, 0.19377568364143372, 0.09343480318784714, -0.16093988716602325, -0.7481544017791748, -2.0153238773345947, -2.4516570568084717, 1.203567624092102, 0.772503137588501, -1.3143556118011475, 1.712141990661621, 0.19064664840698242, -0.09328147768974304, -0.05452299118041992, -1.1874756813049316, 0.4720016121864319, -1.3606352806091309, -0.6370797753334045], [-0.4455939531326294, -0.8042545318603516, -0.9968996047973633, -0.1335381418466568, 1.346902847290039, -0.522664487361908, 2.339191436767578, -1.453940749168396, 1.7359800338745117, 0.09634184837341309, -1.9976578950881958, 0.2555497884750366, -0.15037421882152557, 0.31652122735977173, -0.14571528136730194, -0.5591779947280884, -0.07957256585359573, 0.5704148411750793, -1.1486917734146118, -0.03263389691710472, 0.283066064119339, -1.1961253881454468, 1.7360948324203491, 0.5587741732597351, -0.22713139653205872, -2.375276803970337, -0.0750146433711052, 2.6668906211853027, 0.5919290781021118, -0.5628178715705872, -0.5953052639961243, -1.1277886629104614, 0.5829381942749023, 0.5790450572967529, -0.49859076738357544, -0.6439550518989563, -0.09932250529527664, -0.8882065415382385, 1.0022448301315308, 1.4307416677474976, -0.30198681354522705, 1.9124114513397217, 1.451066017150879, 0.3010737895965576, -0.40974950790405273, 1.393506646156311, -1.6219522953033447, 0.031126560643315315, -0.818152666091919, -0.5392107367515564, 1.1037770509719849, -1.3988429307937622, 0.3470095098018646, 1.547764539718628, -0.7479811906814575, -0.9787521958351135, 2.14506196975708, -1.9010043144226074, -0.03685692697763443, 0.22796276211738586, -1.0232881307601929, 0.2882900834083557, -1.078616976737976, -0.5597847700119019]], [[0.2355152666568756, -0.10466021299362183, 1.018897294998169, 1.1582082509994507, 1.2240198850631714, -2.305635929107666, -0.38370293378829956, -0.4294619858264923, 0.38523873686790466, -1.0892313718795776, -1.3606318235397339, 1.0985685586929321, -1.5069268941879272, -0.4109344184398651, -1.6072925329208374, 0.8558081984519958, -1.0988731384277344, 0.06516758352518082, -0.03437710925936699, 1.5303434133529663, 0.3795037269592285, -1.0078685283660889, -2.1212127208709717, 0.12244363129138947, 0.898246169090271, 2.5808053016662598, -0.5898218750953674, -1.0552802085876465, -0.9017239809036255, 1.8048564195632935, 0.3096015453338623, -0.5847292542457581, 1.2084745168685913, 0.666142463684082, -1.80678391456604, -2.346839427947998, 2.1404495239257812, 0.22265495359897614, -1.0631961822509766, -0.5561016201972961, 1.020684838294983, -0.8692542314529419, 1.166593074798584, 0.15492203831672668, -0.07452018558979034, -0.21600651741027832, -0.907005250453949, -2.2129738330841064, 1.2022264003753662, -0.4867984652519226, -0.49970942735671997, -0.8095400929450989, -0.9354225993156433, -0.6526533961296082, -2.286027431488037, -0.34734123945236206, -0.07636955380439758, -1.228002905845642, 0.2995317578315735, -0.053864043205976486, -0.8011204600334167, -0.7998402714729309, 0.4595440924167633, -0.5411890149116516], [-1.1852163076400757, 0.7220003008842468, -0.19557823240756989, -0.06289451569318771, -0.9077610373497009, -0.039553362876176834, 1.0096153020858765, 0.11880114674568176, 0.3389618992805481, 0.09369229525327682, 0.11920144408941269, -2.2166197299957275, 0.571575939655304, 0.9186254143714905, -0.01948355883359909, 0.7101342678070068, -0.2000073790550232, 1.0655289888381958, -1.45989990234375, 0.4449117183685303, 1.0168888568878174, 1.100903034210205, 0.8990696668624878, 0.6015706062316895, -0.569839596748352, -0.8413155674934387, 0.6515614986419678, -0.46474340558052063, 0.43897736072540283, -1.1306158304214478, -0.018131013959646225, -1.247489333152771, -1.2351044416427612, 0.19654186069965363, 1.4210022687911987, 0.15847769379615784, 1.080812931060791, 0.8133547902107239, 0.4611338973045349, -0.28154513239860535, -0.0057786270044744015, 1.0169602632522583, 1.3641228675842285, 0.261398583650589, 0.6063838005065918, 1.656334638595581, 0.37323710322380066, -0.6746522188186646, -0.6111969351768494, -1.1284406185150146, -1.5450615882873535, -0.3589199185371399, 0.08049242943525314, 0.2774556577205658, 0.5133093595504761, -0.0071625555865466595, 1.681113362312317, -0.8536428213119507, -0.06854093819856644, 1.817050814628601, 1.148462176322937, -0.26195311546325684, 2.700927495956421, 1.128654956817627], [-0.7636023163795471, -0.4000815749168396, 0.9187595844268799, -1.0967029333114624, 1.2058603763580322, 0.8721967339515686, -0.0006915118428878486, 2.3346402645111084, -1.1848078966140747, -0.8852417469024658, 1.0132436752319336, -0.8653731942176819, 1.3483718633651733, -0.7727973461151123, 1.400608777999878, 0.9723018407821655, 0.3057946562767029, -0.631202757358551, -0.7050416469573975, -0.02699524164199829, -2.0698177814483643, -0.4086543917655945, -0.37402039766311646, -0.18614168465137482, -0.9203819632530212, 1.0653752088546753, 0.8015848398208618, -0.4272143542766571, -0.3788924515247345, -1.8399760723114014, -0.36493492126464844, -0.15247312188148499, 1.1425548791885376, 1.3177462816238403, 0.7021995186805725, 0.04053781181573868, 0.9344161748886108, 1.1478631496429443, -0.6211599111557007, -0.7119845747947693, 0.5744881629943848, 0.2982548475265503, 0.0699075236916542, -0.07626360654830933, -1.1729828119277954, -0.00889402162283659, -0.5172264575958252, -0.31898272037506104, -0.5373129844665527, 1.6840155124664307, 0.9934906959533691, 1.4300771951675415, 0.027159690856933594, 1.69199800491333, -0.13598187267780304, -0.13120831549167633, 1.1722607612609863, 0.6599289178848267, 0.15668673813343048, 0.5663874745368958, 1.5526169538497925, 1.3087997436523438, -0.1378554254770279, 0.3017064332962036], [-0.7815080881118774, -0.6397339105606079, 1.2991243600845337, 2.302767515182495, 1.2152118682861328, -0.4426921308040619, 0.9759657979011536, -0.15993165969848633, 0.5114891529083252, 0.6934341788291931, -0.8605181574821472, 0.078328438103199, 1.1562490463256836, -1.0534783601760864, 0.09976594150066376, 1.6745444536209106, 1.1166481971740723, -1.0369526147842407, -0.6111561059951782, -0.49639585614204407, 1.0230598449707031, -1.7828755378723145, -0.044819727540016174, 0.8330972194671631, -0.5081535577774048, 0.4921925961971283, -0.2717403471469879, -1.664930820465088, -1.478983759880066, -0.3359341323375702, 0.2919256091117859, 0.31349557638168335, -0.12405520677566528, 0.06165898218750954, -0.14662940800189972, -1.4545472860336304, 0.6847374439239502, 0.2328488677740097, 0.8717255592346191, 0.9553869366645813, 0.42307719588279724, 1.2786762714385986, 0.5301470160484314, 0.9403446912765503, 1.5291268825531006, -1.4496841430664062, -0.3073219060897827, 0.032451044768095016, 0.11779672652482986, -1.487315058708191, -0.6999922394752502, -1.1227694749832153, 0.995864987373352, 1.1141051054000854, -1.1550238132476807, -0.711844801902771, -0.336789608001709, 0.15347424149513245, 1.8503177165985107, 0.3265121579170227, -1.166179895401001, 0.5598036050796509, 0.4208827018737793, 1.7190502882003784], [-0.35265442728996277, -2.0046067237854004, -0.35208696126937866, -1.6630604267120361, 0.4189176559448242, -1.7192819118499756, -0.35066813230514526, -0.6714525818824768, 1.0108232498168945, 0.2513614296913147, 0.6211129426956177, 0.9662559628486633, -0.35268843173980713, -0.6183351278305054, -0.02415807731449604, 0.8801090717315674, -0.4323929250240326, 0.8845801949501038, -1.1226271390914917, -0.438388466835022, -0.44004005193710327, -1.3304346799850464, 2.8226332664489746, 1.9629794359207153, -2.3611128330230713, 0.626083254814148, -0.4053053557872772, 0.029835712164640427, 1.721416711807251, -1.1703336238861084, -0.8166236281394958, 2.3906311988830566, 0.38040152192115784, -2.1043806076049805, -0.19528444111347198, 0.48794955015182495, 0.2167923003435135, -0.3057870864868164, 1.7521730661392212, 1.7266849279403687, -0.6705009937286377, 0.036823879927396774, -0.006877924781292677, -0.17251266539096832, 0.4004230797290802, -0.03710700199007988, 0.760823130607605, -0.7045655250549316, 0.16393235325813293, 0.7780940532684326, 1.2976521253585815, 0.200094074010849, -0.7134799361228943, 0.05976198986172676, -0.06215853616595268, -0.44169294834136963, 1.0592342615127563, -0.9932650923728943, 0.7908799052238464, 1.2576022148132324, -0.9270119667053223, -2.506868839263916, -1.2900110483169556, 0.3879116475582123], [0.2481730580329895, 0.8019136190414429, 1.4687942266464233, 0.088346928358078, -1.1230177879333496, 1.7948241233825684, -0.15216708183288574, -1.162682056427002, -0.7705379128456116, -3.203611135482788, 0.9346267580986023, -1.3929674625396729, -0.16429947316646576, 0.482909232378006, -1.1580575704574585, -0.0644911453127861, -0.6113926768302917, 0.4281253218650818, 0.4809589087963104, -1.6169774532318115, -1.6387690305709839, 0.6629692912101746, -0.9192995429039001, 0.8788498044013977, 0.5492196679115295, 2.322754383087158, 1.8896843194961548, 1.7840583324432373, 0.013028088957071304, -1.2449047565460205, -0.029295388609170914, 0.8015261888504028, 0.35637539625167847, 0.42130938172340393, -0.6314601302146912, 0.06043488532304764, -0.09862279146909714, -0.1953173279762268, -0.08243486285209656, -0.6456506848335266, -0.07984679192304611, 1.0800100564956665, 0.6984004378318787, 0.5221428275108337, 0.3917410373687744, -0.982560932636261, -0.018626529723405838, 2.104579210281372, -1.1618541479110718, -0.2993517816066742, 1.5047529935836792, 0.9281363487243652, 0.1253480166196823, -0.7288944125175476, -0.25646331906318665, -0.3356490135192871, 1.2996364831924438, -0.7970079779624939, -0.36493757367134094, 1.4390039443969727, -0.6637768745422363, 0.4661581218242645, -0.8585401773452759, 1.9183331727981567], [0.35224252939224243, -0.30565160512924194, -1.3882670402526855, -0.5887367129325867, -1.4267572164535522, -0.6499752402305603, -0.8426721096038818, -0.711051881313324, -0.3614986836910248, -0.7582155466079712, -1.0370843410491943, -0.6709810495376587, -0.1699874997138977, 2.209792137145996, 1.6213496923446655, 0.5729888677597046, 0.06194286048412323, -0.784142792224884, -0.7921195030212402, -0.38309434056282043, 1.39462149143219, 0.36161094903945923, -0.8294634222984314, -0.3021176755428314, 0.23475787043571472, -1.1309995651245117, 1.3844953775405884, 1.1292433738708496, 0.975727915763855, 0.25299009680747986, 0.33454641699790955, -0.3347227871417999, 1.4940191507339478, 1.4967041015625, -1.3033537864685059, 0.004435290582478046, 1.4064921140670776, -0.31878677010536194, 0.10556435585021973, -1.37967050075531, 1.6942685842514038, 0.21903890371322632, -1.2424516677856445, -0.08696030080318451, 0.41935545206069946, 1.6877307891845703, -0.3518943786621094, -1.5901230573654175, -0.21932180225849152, 0.5374293923377991, 0.6455644369125366, 0.2535060942173004, 0.571008563041687, -0.11645543575286865, 0.6507523655891418, 1.7083611488342285, -0.4350177049636841, 0.7876835465431213, 0.9726076126098633, -0.785308301448822, 0.5665773153305054, -0.4676898121833801, -1.350888729095459, -1.6777900457382202], [1.1495285034179688, -2.280667543411255, -0.18445193767547607, -0.8819329142570496, 0.2240435928106308, -0.37251365184783936, -0.04589613527059555, 0.1941089630126953, -0.977929949760437, -0.206051304936409, 1.690765619277954, -1.927566647529602, 0.3702256977558136, 0.9883744120597839, -0.7234342694282532, 1.5725315809249878, -0.26481881737709045, 1.2102996110916138, 2.0049028396606445, -0.7054837942123413, -0.3166714906692505, 0.5890660285949707, 0.5254777669906616, -1.9184424877166748, 0.28894588351249695, 0.8225407600402832, 0.7188402414321899, -0.565227210521698, -0.6515454053878784, -0.4148511290550232, -0.7333202958106995, -0.5135535597801208, 0.6856685876846313, -0.5152629017829895, 0.05914829298853874, 0.534852147102356, -0.39410874247550964, 0.06640651822090149, 0.1765027940273285, 3.7462871074676514, 0.10328029841184616, -1.3579996824264526, -0.5311200022697449, -1.8948370218276978, -1.2425585985183716, -0.12817566096782684, -1.884141206741333, 0.11197452992200851, -0.32749056816101074, 0.15679684281349182, -0.13132429122924805, 1.2254596948623657, 0.7558699250221252, -1.440569281578064, 0.6528547406196594, -0.9487615823745728, -0.5123181343078613, 0.3117050528526306, -0.27352651953697205, -1.7095296382904053, -1.4650919437408447, -0.19747252762317657, 1.0475482940673828, -0.16150735318660736], [0.5050193071365356, -0.42899179458618164, -0.012818953022360802, 0.011504920199513435, -0.8950791954994202, 0.7143340706825256, -0.5646110773086548, 0.9212552905082703, -0.38817551732063293, -0.2031262218952179, -0.0954568013548851, 0.19314981997013092, -0.137056365609169, 0.7592709064483643, -0.4875294864177704, 0.43236619234085083, 0.2931496500968933, -0.7610003352165222, -0.027659201994538307, 0.5244830250740051, 0.6689106822013855, 1.4425767660140991, 0.16226765513420105, -1.1880340576171875, 1.1668545007705688, -0.11634636670351028, 0.517680823802948, -0.733701229095459, 0.5591639280319214, 2.9635815620422363, -0.0349343977868557, 0.30032408237457275, -0.7179433107376099, 0.005347475875169039, -0.1697390079498291, -0.17198893427848816, -1.3562151193618774, -2.0751237869262695, 0.5058648586273193, -0.5651516318321228, -0.1718706339597702, 1.0086256265640259, 1.4777899980545044, 0.4495755136013031, -0.3590601682662964, 1.1205854415893555, 1.108406901359558, 1.1962984800338745, -1.698279857635498, -0.33790090680122375, -0.7600352168083191, -0.0470944419503212, -0.23024195432662964, 0.14464271068572998, 0.14293377101421356, 0.13588353991508484, 0.41819581389427185, 0.23910556733608246, 1.2475374937057495, 0.38230326771736145, -0.06756841391324997, -2.2658605575561523, 0.8525826930999756, 0.2471274733543396], [-0.03075227700173855, -0.7283056378364563, -0.06388308852910995, -0.3296146094799042, -1.8181040287017822, -0.3582853376865387, -0.004560456611216068, 0.5669193267822266, 0.5201267004013062, -0.2672313153743744, -0.054499488323926926, -0.4350196123123169, 0.5751049518585205, 0.37696805596351624, 0.08618863672018051, 1.4837883710861206, -0.22044780850410461, -0.22564300894737244, -1.0395375490188599, -1.6837308406829834, 0.07842221856117249, 0.6487431526184082, 1.4573014974594116, 0.5262570977210999, 0.07623102515935898, 0.26573267579078674, 1.749672293663025, -1.8540595769882202, 0.539430558681488, 0.7239049077033997, -0.46691200137138367, 1.5533430576324463, 0.4688219130039215, 0.2527543306350708, 0.010485775768756866, -1.7801593542099, 0.2780853509902954, -0.5115019083023071, 2.648609161376953, -0.908113420009613, 1.216646671295166, 1.3794232606887817, -0.5539999604225159, 0.19519011676311493, 0.6792540550231934, 0.2985866069793701, 0.7328551411628723, -0.9105565547943115, 1.622247576713562, -1.5166912078857422, 0.008435684256255627, -0.9346024990081787, 0.8483893871307373, 0.10429757833480835, -1.3613625764846802, 0.09186800569295883, -0.06933252513408661, 0.08150701224803925, -1.652418851852417, -0.7512048482894897, -0.02225092239677906, -0.5508002042770386, 0.7781292796134949, -2.0173637866973877], [-1.2362630367279053, -0.7839424014091492, 0.5346019864082336, 1.1146880388259888, -1.1201361417770386, 1.0688623189926147, -2.2410802841186523, -0.44269052147865295, -0.11167246103286743, 1.1365247964859009, 0.30417266488075256, -0.49903473258018494, -1.1720595359802246, 2.3471062183380127, -0.0023692382965236902, -1.4820362329483032, -0.3596392869949341, -0.3941253125667572, -1.3192769289016724, 1.6290634870529175, -0.38741034269332886, -0.5019455552101135, 1.5573312044143677, 0.9429478645324707, -1.466333031654358, -0.25475722551345825, 1.661340594291687, -0.8475913405418396, -0.7126779556274414, -1.7312828302383423, -0.8935274481773376, -0.008323295041918755, -0.47029051184654236, -0.32477623224258423, 0.31105127930641174, -0.07310274988412857, 0.6721614003181458, -0.3593936562538147, -1.741929054260254, 0.5425547361373901, 0.33885496854782104, -0.7054768204689026, -0.5643675923347473, 1.357082486152649, 0.6447165608406067, 0.4230227768421173, -0.6484017372131348, -0.886019766330719, -0.08373931795358658, -0.20803861320018768, 0.8945404887199402, -0.2093065232038498, 0.711242139339447, 1.3761060237884521, -1.9232115745544434, -0.225655660033226, 0.35879844427108765, 0.900184690952301, 0.9589802622795105, 0.318418949842453, 1.2086454629898071, -1.0038362741470337, -0.49217748641967773, 0.4536602795124054], [-0.6450989246368408, -0.40607768297195435, 1.7140796184539795, 0.19571521878242493, -0.021466312929987907, -2.1355245113372803, 1.5675420761108398, 0.5016597509384155, 0.17885151505470276, 0.3287722170352936, 0.40979334712028503, 0.3689192831516266, -0.7346307635307312, -1.475390076637268, 0.059034135192632675, 0.6832982301712036, -0.4764159321784973, -1.0408638715744019, 0.6437610387802124, -0.47490057349205017, -0.5786759257316589, 0.22058378159999847, 0.44708624482154846, -0.45717573165893555, -0.15838390588760376, -0.5290434956550598, -0.3796811103820801, -1.3916388750076294, -0.5394350290298462, 0.13712136447429657, -1.2523773908615112, 0.34221532940864563, -2.4640588760375977, 0.41912585496902466, 0.02429136075079441, -0.2040688842535019, -0.11484963446855545, -0.12170133739709854, -0.15936098992824554, 0.47947996854782104, -0.04216678813099861, 0.5619974136352539, 0.4339981973171234, -0.21225818991661072, 0.33997803926467896, 1.4102373123168945, 1.3056108951568604, 0.5827361941337585, -0.27992168068885803, -0.3503814935684204, -0.7396354675292969, 0.21085642278194427, -0.1729731261730194, -0.5056062340736389, -2.0189225673675537, 0.6051895022392273, -0.7771638631820679, 0.6152321696281433, 0.5374760627746582, 0.06952500343322754, 0.5101101994514465, 0.013681571930646896, 0.2614356279373169, 0.9345070123672485], [1.0036311149597168, -1.1893110275268555, -0.6094391942024231, -0.7335831522941589, 1.8046587705612183, 0.4233887493610382, -0.435857892036438, 0.40918293595314026, -1.763843297958374, 0.24183601140975952, 1.2419878244400024, -0.5490918159484863, 2.2509024143218994, 0.21209418773651123, -1.998389720916748, 1.24061119556427, 0.5311489701271057, 0.14085620641708374, 0.9320411086082458, -1.6973508596420288, -0.9744842052459717, -1.1783162355422974, -1.0987657308578491, 0.17970231175422668, -1.1525405645370483, -0.11741054058074951, 1.459389567375183, -1.2994853258132935, 1.2811726331710815, -0.03780724108219147, -0.7378501296043396, -1.7470027208328247, 1.1084916591644287, -0.1871887743473053, 0.5953169465065002, 2.185856580734253, -0.04848825931549072, -0.17810751497745514, 0.510911226272583, 0.7639049291610718, 0.23784103989601135, 0.6513977646827698, 0.4446111023426056, -0.15202315151691437, 0.2125740647315979, -2.061591863632202, -0.11736010015010834, 1.7085201740264893, -1.231358528137207, 0.31969428062438965, 2.2279865741729736, -0.403168261051178, 1.248816967010498, -0.24281658232212067, 0.08176281303167343, 0.9446675181388855, 1.3564056158065796, 0.07493522018194199, 0.1527075171470642, 0.017587868496775627, -0.10199666023254395, 0.8635730147361755, 1.2645801305770874, -0.4500119686126709], [0.8938802480697632, -0.21351832151412964, -1.0124115943908691, 0.9032676815986633, 1.1691336631774902, 0.2776249945163727, -2.4873268604278564, -1.1759724617004395, -0.18642841279506683, -0.13147835433483124, -0.5494755506515503, 1.8411405086517334, 0.5464442372322083, -0.5269930362701416, -0.1340171992778778, 0.715042233467102, -0.22194306552410126, 0.4480884373188019, 0.06042398139834404, -0.380185604095459, 0.5258457064628601, -0.7782179713249207, 0.3793010711669922, 1.6088671684265137, -0.9149744510650635, -1.7952697277069092, -1.2138651609420776, 0.8342794179916382, 1.2457205057144165, -0.39201197028160095, -1.5240355730056763, 0.9329840540885925, 0.7581905126571655, 1.164838433265686, 1.3388510942459106, 0.2912100553512573, 0.09656645357608795, 0.7458353638648987, -0.3530625104904175, -0.09672144055366516, -1.4016377925872803, -1.2599478960037231, -0.9648175835609436, 0.33155563473701477, 0.09652943909168243, 0.5067187547683716, -0.7637730836868286, 1.016149878501892, -0.7018784880638123, 0.4624287486076355, 1.499887228012085, -1.7926414012908936, -0.2934715151786804, 1.0812489986419678, -1.2361418008804321, -0.26186197996139526, 0.21480201184749603, -0.41425591707229614, -1.102096676826477, -1.3452072143554688, -0.18648676574230194, -0.06311892718076706, 0.4425467550754547, -0.658406674861908], [0.08517380803823471, -0.6672956943511963, -0.9488962888717651, -0.918231189250946, 1.0010746717453003, 0.6985029578208923, -0.5164437890052795, -0.9004988074302673, 1.7931090593338013, 0.8097476363182068, 0.6392858028411865, -1.374197244644165, 0.48916101455688477, -1.779259204864502, -0.00495326379314065, -0.7840290069580078, -0.5662937760353088, -2.0837912559509277, -2.616384744644165, 1.9870479106903076, -0.8051345944404602, -1.7109930515289307, -0.9351174235343933, 0.24303178489208221, 0.3660532236099243, 3.0032811164855957, -1.5214250087738037, -1.317401647567749, -1.0468637943267822, 0.10396909713745117, -0.8832909464836121, -1.2989553213119507, 0.1023198738694191, 0.11284578591585159, 1.4278358221054077, -0.7065039873123169, -0.8503172397613525, -0.021669358015060425, 0.19959446787834167, -1.0285242795944214, 0.516559362411499, -0.8485429286956787, -0.6614418625831604, -0.2807236611843109, -1.6923936605453491, -0.30464404821395874, -1.2771074771881104, -0.4098384976387024, -0.3285049796104431, -0.46341875195503235, 0.48273611068725586, 0.6540045738220215, 0.43536561727523804, -1.5498424768447876, 0.6216974854469299, -0.3803464472293854, 0.2519225776195526, -0.633558452129364, 1.7574831247329712, 1.0682469606399536, -0.982076108455658, -0.18745861947536469, 0.8883086442947388, 0.8633409142494202], [0.6609224677085876, 0.8467981219291687, -0.3080574572086334, 0.5401238203048706, 0.15196342766284943, -0.7358066439628601, -0.9339752793312073, 0.6294453740119934, 1.1387075185775757, -1.4808529615402222, 1.0722105503082275, 0.5030590891838074, -1.1106046438217163, -0.5401252508163452, 0.214365154504776, -1.1244714260101318, 1.0651721954345703, -1.245999813079834, -1.3482511043548584, 0.48705166578292847, 0.975320041179657, 0.01705063506960869, 1.082202672958374, 1.081252098083496, -0.8801530003547668, -0.18475545942783356, -1.288718581199646, -0.37309929728507996, 0.09388475865125656, -0.4156383275985718, 0.4410379230976105, 1.6862342357635498, 0.5609003901481628, 0.9252389073371887, 1.5535553693771362, 0.2255823165178299, 0.0924518033862114, 0.7105473875999451, -0.6388722062110901, 0.07624347507953644, 1.918531894683838, 0.4418470859527588, -0.10132946819067001, -0.2155132293701172, -0.7164098620414734, 0.638592541217804, -0.7072252035140991, -0.1969221532344818, 0.8838042616844177, -1.3381274938583374, -0.3568665385246277, 0.8409976959228516, -0.7613650560379028, -1.6123957633972168, 0.8349620699882507, -0.06892051547765732, -0.3536379337310791, -1.1833103895187378, 0.32228749990463257, -0.46550044417381287, 2.014253616333008, -0.29783958196640015, 1.0947548151016235, 1.834128975868225]]], [[[-0.4762415885925293, -1.1294341087341309, -0.21247495710849762, -0.42898041009902954, -0.8476241827011108, 0.2690390646457672, 0.49128231406211853, -1.7129861116409302, -0.7032782435417175, 1.1874048709869385, -0.13850142061710358, -0.8435003757476807, 0.5370597243309021, 0.3311793804168701, -1.5634294748306274, 0.40668338537216187, 0.4036882817745209, -1.4357783794403076, 0.5905762314796448, -0.23503117263317108, 0.3000231087207794, -0.10193075239658356, 0.19224029779434204, -1.3398174047470093, 0.6052118539810181, -1.8104068040847778, 0.4195577800273895, 0.6417385339736938, 0.7185598015785217, 1.859444260597229, 0.4561360776424408, -1.6145989894866943, -2.3203251361846924, 0.7940852046012878, 0.8595538139343262, -1.0104339122772217, 0.11108958721160889, -0.8131095767021179, -0.9745880961418152, -1.0890123844146729, 0.1905294507741928, 0.7711371779441833, 0.03747270256280899, -1.0958373546600342, 1.5862997770309448, -0.9303790330886841, -1.9091315269470215, 0.6627694368362427, -0.4248737096786499, 0.628926157951355, 0.6409845948219299, -0.16213923692703247, 0.8124426603317261, 0.45415735244750977, 0.22667072713375092, -0.5915957093238831, -0.647099494934082, 0.6921771764755249, 0.5393890142440796, -1.1179965734481812, 0.8633712530136108, -0.7922371625900269, -0.22585278749465942, 0.7980647683143616], [-0.3862917423248291, 0.7439643740653992, -0.16072995960712433, 0.6099331974983215, 0.0798906534910202, 0.08328339457511902, -0.7825604677200317, -0.761468231678009, 0.49353960156440735, -1.7980386018753052, 0.8130020499229431, 0.20971760153770447, -1.1610219478607178, 2.2991445064544678, 0.7003118991851807, -0.7059206366539001, -0.5815931558609009, 2.0905230045318604, -0.3264305889606476, 1.581035852432251, -0.3330623209476471, -0.41465049982070923, 0.8907473683357239, 1.471864104270935, 1.4099483489990234, -0.34973177313804626, 0.15611112117767334, -0.46167412400245667, -0.7871654629707336, 0.6582253575325012, 0.04616600275039673, 0.4215374290943146, -0.09934835135936737, -1.7085247039794922, -2.2827117443084717, -0.45034098625183105, -0.6211868524551392, 1.7170368432998657, 0.013188255950808525, -0.2699723243713379, 2.5362586975097656, 1.4395583868026733, 1.2122834920883179, -0.004815963562577963, -0.2411622703075409, 0.9320635199546814, -0.1740390807390213, 0.7321662306785583, 0.26024025678634644, -1.0914804935455322, 1.0510380268096924, 1.019715666770935, -1.8812322616577148, -1.0149978399276733, -0.3046368658542633, -1.446871042251587, 0.31546878814697266, 1.3865612745285034, 0.4332870543003082, -0.8467918634414673, 1.4929189682006836, 0.40209710597991943, 1.0653223991394043, -0.26222649216651917], [0.143632709980011, 1.5306618213653564, -0.016098273918032646, 1.428079605102539, -1.4361473321914673, 1.7034357786178589, 0.4895586371421814, -0.1972586065530777, 0.008450893685221672, -1.3310548067092896, 0.1663265824317932, 0.6423986554145813, 0.05844651535153389, 0.6274150013923645, 1.6686984300613403, -0.8892531394958496, 0.9662182927131653, 0.7164018750190735, -0.15893635153770447, 0.27579665184020996, 0.21471264958381653, 0.42408785223960876, -2.3359129428863525, -1.7102489471435547, 0.7719685435295105, -0.20924833416938782, 1.5480539798736572, 0.4066024720668793, 0.23276931047439575, -0.5396002531051636, 0.34889236092567444, -0.5868417620658875, 0.32107436656951904, 1.0014832019805908, -0.11186414211988449, -0.8859967589378357, 1.4647654294967651, 0.4268152713775635, -0.015542721375823021, 0.10455360263586044, 0.4014490246772766, -0.03947460278868675, -0.33685579895973206, 0.6683758497238159, -1.236083745956421, 0.40314093232154846, -0.4179278314113617, -0.70651775598526, -2.425980567932129, -0.02831961400806904, 0.8550817370414734, 0.8314409255981445, 0.15834493935108185, 0.08667359501123428, -1.3279775381088257, -0.27391576766967773, -0.41879427433013916, -0.31057482957839966, 0.6046705842018127, 0.5571097135543823, 0.9412106275558472, -1.9155558347702026, 0.33501243591308594, -0.12494775652885437], [0.30205732583999634, 1.559950590133667, 1.2861182689666748, 1.073791742324829, 0.6485075950622559, -0.14932069182395935, 1.0979597568511963, 0.38591647148132324, 0.1543866991996765, 0.4040150046348572, 0.5579670667648315, 0.7880021333694458, -0.43859124183654785, 0.34455612301826477, -0.10012178122997284, -1.6611989736557007, -1.1415984630584717, -0.30855080485343933, 0.4611462354660034, -0.7240217924118042, -0.4642195701599121, 0.2372167855501175, -1.0328267812728882, -1.0501829385757446, -1.407860279083252, 0.48047494888305664, 0.828476071357727, -0.13500449061393738, -0.9874749779701233, -1.370201826095581, -0.5215896368026733, 0.26811128854751587, 1.720645546913147, 0.0847875326871872, -1.1363400220870972, 0.5448320508003235, 1.6066285371780396, -0.3575778901576996, -1.4249018430709839, -0.8205471038818359, -0.5304379463195801, -0.31792759895324707, 2.5078554153442383, 0.4547836184501648, -0.2816382944583893, -0.47036072611808777, 0.8247971534729004, -0.888526201248169, -0.7356468439102173, 0.6751782894134521, 1.3900384902954102, -1.3126871585845947, -0.3445444107055664, 0.2549538314342499, 0.0863061398267746, 0.11701846122741699, -0.26954635977745056, -0.4984169900417328, 1.6758289337158203, -0.008772416971623898, -2.747875213623047, -1.8449991941452026, -0.0802355408668518, 0.29390421509742737], [-1.723604679107666, 0.32435110211372375, 0.5317571759223938, 0.2557806074619293, -1.6291117668151855, 1.0580755472183228, 0.21237513422966003, 0.3455091416835785, -0.5481398701667786, 0.87593674659729, 0.958031415939331, -0.9183218479156494, -1.063617467880249, -0.3847521245479584, 0.24255257844924927, 2.3083078861236572, 1.6457295417785645, -0.6962448954582214, -0.8207870125770569, 1.1933016777038574, -0.8854057192802429, -0.5937576293945312, -1.0168501138687134, -2.320180892944336, -0.6084939241409302, -1.0584784746170044, 0.5258122086524963, -0.6521655321121216, 0.3272555470466614, -1.3291211128234863, -0.4645509123802185, 1.7752166986465454, 1.8222780227661133, 1.0957294702529907, 0.7168442606925964, 1.1412544250488281, -0.24856425821781158, 1.9361367225646973, -1.1124134063720703, 0.7853408455848694, -0.23620261251926422, -0.693448543548584, 1.7908108234405518, -0.19524356722831726, 0.003998965956270695, -1.3709681034088135, -0.945660412311554, 0.5859221816062927, 1.0432722568511963, 1.3299570083618164, -0.0692039281129837, -0.13148801028728485, -0.8448122143745422, -0.3153878450393677, 1.1202484369277954, 0.7685649394989014, 0.6582436561584473, 0.39151668548583984, -0.911592423915863, 1.3432942628860474, -2.370666742324829, 0.6218851208686829, -0.604127824306488, 0.8472200036048889], [-0.28677815198898315, -1.0417219400405884, -0.1925656944513321, 1.218941569328308, 0.41332077980041504, 1.8973381519317627, -1.2058589458465576, -0.13956525921821594, 0.3007836937904358, -1.0207626819610596, 0.33607175946235657, -0.35614386200904846, 1.1699098348617554, 1.3759185075759888, -0.4191552400588989, -0.1243908479809761, 0.35463830828666687, -0.0124129056930542, -0.4547426700592041, -0.01268650684505701, -0.18948213756084442, 1.1406418085098267, 0.24661171436309814, -0.04401659220457077, 2.0610852241516113, 0.2509840428829193, -0.9167769551277161, -0.22323380410671234, 0.22592468559741974, 0.6270700693130493, 1.1356302499771118, -1.1813513040542603, -1.0789328813552856, -1.28115975856781, -0.12573328614234924, -1.5918993949890137, -0.8885741233825684, -0.09836045652627945, -0.3136981129646301, -0.5207041501998901, 0.5640081167221069, -0.15168772637844086, 1.0001767873764038, 0.3849373459815979, 0.32138559222221375, -0.1464228630065918, -0.48383358120918274, -0.055447883903980255, -0.3583614230155945, -1.9785770177841187, 1.360102891921997, -1.778670310974121, 0.6426130533218384, -1.2059820890426636, -1.5556583404541016, 0.35200774669647217, 1.3340176343917847, -0.729513943195343, 0.857189953327179, 1.2438385486602783, -1.5137661695480347, -0.32849475741386414, -0.5322401523590088, 1.1194757223129272], [0.9658584594726562, -0.6751036047935486, -0.5098749995231628, 0.9604613780975342, -0.7421318888664246, -0.12002070248126984, 0.48861128091812134, -0.0715683251619339, 0.8784609436988831, 1.2389569282531738, -0.42734524607658386, -0.7081524729728699, 0.33672651648521423, -0.3365502953529358, -0.27660053968429565, 0.8293609023094177, -0.6356480121612549, -0.632666289806366, -0.1256788969039917, -0.33052605390548706, 0.9723623394966125, -0.8063360452651978, 0.26080843806266785, 0.22592607140541077, -0.7011193037033081, 0.949497401714325, -1.6515504121780396, 1.3007862567901611, -0.3633144199848175, 1.0082752704620361, 0.16545046865940094, -2.294628381729126, -1.548762559890747, 0.07718274742364883, 0.6264434456825256, 0.9618324041366577, 0.6013339757919312, 1.0902822017669678, 0.319713830947876, 0.6904246807098389, 0.8293899893760681, 0.989339292049408, -1.7681350708007812, -0.6711991429328918, 0.03544631972908974, -2.023785352706909, 1.6273491382598877, 1.853161334991455, -1.325148105621338, 0.27561673521995544, 0.3855971693992615, 1.678254246711731, -0.95377516746521, -0.17399999499320984, -0.2985082268714905, 0.5111009478569031, -0.6698442101478577, -0.5877992510795593, -0.9608643054962158, -0.46770498156547546, -1.2834140062332153, -1.3173820972442627, -0.21646827459335327, 0.4247101843357086], [1.8440335988998413, 0.31369614601135254, 0.004935725126415491, -0.93315190076828, -0.6958403587341309, 1.137520670890808, 0.35382628440856934, 0.04385925829410553, 1.7597851753234863, 0.4795789122581482, 0.2006072849035263, -1.5750908851623535, -0.06546543538570404, 0.9234901070594788, -0.07217028737068176, -2.9055230617523193, 0.24730825424194336, -1.2901413440704346, -1.8710249662399292, 1.038670301437378, -0.7922561168670654, 0.2113012969493866, -2.575920820236206, -0.25351908802986145, 1.3919432163238525, -0.8807618021965027, -0.08736657351255417, 0.06848340481519699, 0.019268836826086044, -0.5230552554130554, -1.7033607959747314, -0.7273638248443604, 2.7893178462982178, 0.48555848002433777, 0.6625788807868958, -0.6081266403198242, 1.316060185432434, 0.1954440176486969, -1.2088415622711182, -1.5025930404663086, -0.07431351393461227, 0.5113115906715393, -0.0031972010619938374, -2.0577023029327393, 0.08627954125404358, 0.069468192756176, -0.8043855428695679, -1.7568104267120361, 0.037218254059553146, -0.823927104473114, -0.3452894687652588, -0.44319602847099304, 0.007813596166670322, -1.3258168697357178, -1.2225161790847778, 3.0751595497131348, 1.1606489419937134, -0.4211746156215668, 0.421008437871933, 0.6251091957092285, 0.057643450796604156, 0.16526901721954346, -0.005918701179325581, -1.1758065223693848], [0.10166596621274948, -0.03705182299017906, -0.5845822691917419, -3.0685813426971436, -0.6174756288528442, -0.850609302520752, -1.2262108325958252, -0.48311781883239746, -0.5670138001441956, 0.7630916237831116, -1.2396982908248901, -1.6077989339828491, 1.4142805337905884, -0.5687341094017029, -0.27396735548973083, -0.8039740324020386, 1.517831802368164, -0.552257239818573, 1.4669610261917114, 0.6850219964981079, -1.0561978816986084, -0.5002845525741577, -0.7253217101097107, 0.8995784521102905, 0.7391057014465332, -1.0004256963729858, 1.4866036176681519, -0.39933642745018005, -0.05713127180933952, 0.8989756107330322, -0.7438763380050659, -0.3345828652381897, 1.3975491523742676, 0.227035790681839, 1.3320515155792236, -0.16710717976093292, 0.02397344820201397, 1.1490973234176636, 0.8759214878082275, 0.6438634395599365, 1.0797854661941528, 1.1847952604293823, -0.6626052260398865, 1.7174837589263916, 1.2196929454803467, -0.37052974104881287, 1.8966296911239624, -0.03482028841972351, 0.145647794008255, 0.2433149665594101, -0.29262736439704895, 1.0897963047027588, 0.15232127904891968, -0.46935200691223145, -0.6761954426765442, -0.2671128511428833, -0.2204882949590683, -0.37149637937545776, -0.10110887885093689, 0.047425027936697006, 0.05803623050451279, 1.5216599702835083, -1.2568222284317017, 0.21738825738430023], [-1.357856273651123, 0.4645688533782959, 1.153469204902649, -0.8700218200683594, 0.8766265511512756, -0.4242557883262634, 1.1897310018539429, -0.11918331682682037, -0.5133398771286011, 0.26278355717658997, -0.5568469762802124, 0.1768079698085785, -0.42632657289505005, -1.12477445602417, 0.16196423768997192, -0.6732879877090454, 0.5422975420951843, -2.2567358016967773, 0.49037376046180725, 1.5778157711029053, 0.8995856642723083, -0.5922080278396606, 2.260495662689209, -0.7389471530914307, -0.4260227680206299, -1.4864888191223145, 0.3556804358959198, 1.2065757513046265, 0.2582765519618988, -0.3066854774951935, 0.45012718439102173, 1.8306885957717896, -0.33616557717323303, -0.6081551313400269, 0.8971949219703674, 0.6376999616622925, 0.4178880751132965, 1.1323450803756714, 1.2292892932891846, 0.19850090146064758, 1.4636200666427612, 0.2851715087890625, 0.5078747868537903, -1.2333729267120361, 0.7668204307556152, 1.6054571866989136, 2.0069189071655273, 0.1948077380657196, 1.2074865102767944, 0.8287904858589172, -0.8990893363952637, 1.1685154438018799, 1.667115569114685, -0.3729262351989746, -2.3348662853240967, 0.060719314962625504, 0.10121111571788788, 0.3301346004009247, -0.1580895483493805, 0.09550297260284424, -0.3777310252189636, 1.6864619255065918, 0.26679953932762146, 0.0056963274255394936], [1.159936785697937, 0.5093096494674683, 1.094188928604126, 0.001582933240570128, -1.244504690170288, 1.3681164979934692, -0.5618487596511841, -0.19223305583000183, -0.5430223941802979, 0.0007286079344339669, -0.5604934692382812, -0.4527803361415863, -0.8770367503166199, -0.1989872306585312, 0.10009349137544632, -0.3217376172542572, -0.8913240432739258, -0.3871554136276245, 0.44998791813850403, -0.41296350955963135, 0.6021215915679932, -0.7129284739494324, 0.8341543674468994, 0.6412782073020935, -1.363153100013733, 1.0759997367858887, -0.004521167371422052, 1.536157250404358, 0.4214830696582794, 2.218536138534546, -0.12124619632959366, 1.7567181587219238, 0.5056160688400269, 1.914907455444336, -2.127030849456787, -1.5661568641662598, 0.32631054520606995, -0.977749228477478, 0.04054897278547287, -1.9019020795822144, -0.3933776319026947, -1.1248736381530762, -1.5151065587997437, 0.08564125746488571, 1.060925006866455, -0.44387465715408325, -0.6630865335464478, 1.4887621402740479, 0.7375043034553528, -0.43229836225509644, 0.7972471714019775, 0.6435408592224121, -0.9106280207633972, 0.7571913599967957, -1.6907764673233032, -0.4833317995071411, 1.076317310333252, -0.11018919944763184, -1.8363146781921387, 0.4409528374671936, -0.3000720739364624, 0.058090660721063614, 0.35573282837867737, 0.04187316447496414], [0.14524568617343903, -0.642185389995575, 0.8101137280464172, 0.9152015447616577, 0.522257924079895, -1.7012395858764648, -0.7235016822814941, 0.19435696303844452, -0.129092738032341, 0.07468587905168533, -0.5676997900009155, -0.800080418586731, -0.8425219058990479, 0.6163857579231262, -0.511121392250061, 1.2098175287246704, 0.11697308719158173, -0.1695384681224823, -1.5445420742034912, -0.043668828904628754, 2.932023763656616, -1.6199544668197632, -1.4393125772476196, 0.1328403353691101, 0.23108476400375366, 2.0288784503936768, -1.1050161123275757, 0.9563701152801514, -0.042367007583379745, -1.4035325050354004, -0.7240862846374512, -0.6174461841583252, -1.3713726997375488, -0.6203316450119019, 0.17085573077201843, 0.9227756261825562, 1.9984066486358643, -1.1508252620697021, -0.026624029502272606, 0.00667658681049943, 1.4668930768966675, -1.009785532951355, 0.027655145153403282, -0.4879920780658722, 0.04027874022722244, 0.029071036726236343, 0.04444678872823715, 1.191707968711853, 0.9147062301635742, 0.303943008184433, 2.8593411445617676, 2.1134707927703857, 2.088339328765869, -0.3838246464729309, -0.24981889128684998, 2.458608865737915, -1.1768428087234497, 0.310346782207489, -0.11239097267389297, 1.8413317203521729, 0.6471856832504272, -0.14895811676979065, 0.5357813835144043, -0.7011310458183289], [0.6374489068984985, 0.032515749335289, 0.2301243394613266, -0.4914436638355255, 1.7660959959030151, -0.9013055562973022, -0.773421049118042, 0.3440929651260376, -0.14628159999847412, 0.3290520906448364, -0.12166589498519897, 1.4543037414550781, -0.12623412907123566, -0.10673730075359344, -0.6792982220649719, 0.6419416666030884, 0.1418379247188568, 0.6416435837745667, 2.048102855682373, 0.01494115125387907, 1.3492604494094849, -0.4058651328086853, -0.7787484526634216, 0.6092599630355835, 1.5161385536193848, -0.2024347186088562, -0.22748486697673798, -2.487401008605957, 1.6103267669677734, -1.1822941303253174, -1.2483432292938232, 0.1648050993680954, 0.7052446603775024, 0.6913759708404541, 1.1654400825500488, 0.9405807852745056, -0.10978706181049347, -0.8567031025886536, -1.4182636737823486, 0.9224109649658203, 0.9882210493087769, -0.9501656293869019, 0.4396805167198181, 1.5402873754501343, -0.31569382548332214, 0.038880810141563416, -1.0277788639068604, -0.04979966953396797, 0.4190645217895508, 0.1516486257314682, 1.075283408164978, 1.5368289947509766, -0.7402275204658508, -0.22768595814704895, -0.09488211572170258, 0.7065293192863464, -0.17055176198482513, -0.5008233189582825, 0.08113963901996613, 1.440895676612854, -0.9600112438201904, -0.4512940049171448, 0.2568536102771759, 1.3226277828216553], [-1.1801069974899292, -0.6796995997428894, -0.7253657579421997, -0.7635452151298523, -1.6574103832244873, 0.0376807264983654, 2.279564380645752, -2.08799409866333, 0.04932313412427902, 0.314167857170105, -0.4574904143810272, 0.8639589548110962, 0.4432869255542755, 0.5172260999679565, -0.7686432600021362, 1.049048900604248, -0.32830190658569336, 2.515613079071045, 0.5546019673347473, 0.12956503033638, 2.0545570850372314, -1.8055285215377808, 0.23111195862293243, 0.005832672584801912, 0.6589149236679077, -0.8622723817825317, -0.21258410811424255, -0.9239156246185303, -0.11116600036621094, 1.1108330488204956, -0.4301149249076843, -0.3815292716026306, 0.2521003186702728, 0.6116729974746704, 0.7084041833877563, -1.110129952430725, 0.7150770425796509, 0.20917484164237976, -0.8827512264251709, 1.012518048286438, -1.181087613105774, 0.4914892911911011, -0.9167470932006836, 0.7177589535713196, -2.614999294281006, -0.451043039560318, -0.019574785605072975, -0.040307652205228806, -1.5167073011398315, 0.794614315032959, 0.5344135761260986, 1.1222282648086548, -0.960464596748352, 0.061865128576755524, 0.6370879411697388, 0.0978793129324913, 1.054306983947754, 0.61850506067276, -0.796292245388031, 0.7245662808418274, -0.8640963435173035, 1.5059314966201782, 0.1548122763633728, 1.8584449291229248], [-0.47223398089408875, -0.9587969779968262, 0.32471075654029846, 0.3600781261920929, 0.21759697794914246, 2.2114334106445312, 0.6290370225906372, -1.3904204368591309, 0.8716539740562439, -0.13345499336719513, -1.2194160223007202, 0.9154781699180603, -1.2049425840377808, -1.9696654081344604, -0.6567336916923523, 0.7505349516868591, -0.19487011432647705, -0.970435380935669, -0.4420679211616516, -0.4861636459827423, 1.4987263679504395, -1.3743783235549927, 0.3720734715461731, 0.5076568722724915, -1.0427683591842651, -0.7643014788627625, 2.3537609577178955, 1.1975377798080444, -0.6954118013381958, -0.5836452841758728, 0.8956634402275085, 1.1820095777511597, -1.15947425365448, -1.9039100408554077, -0.363821417093277, -0.7066891193389893, -0.6354709267616272, 0.5854835510253906, -0.7891553044319153, -0.40942031145095825, -0.06446204334497452, -0.19030949473381042, 0.9565212726593018, 2.393430709838867, -0.965359628200531, 0.40946316719055176, 0.45087307691574097, 0.7144511938095093, 0.38350221514701843, -0.6625354290008545, 0.36707162857055664, -0.540052592754364, 0.00720382621511817, 0.8380831480026245, -1.5496947765350342, -1.0660455226898193, 2.116636276245117, 0.3114974796772003, 2.1135916709899902, -0.26304781436920166, -0.5904996395111084, -1.157274603843689, 2.4030609130859375, -0.4643895626068115], [-0.2357594221830368, 0.20519529283046722, 0.4709007143974304, 2.453434705734253, 0.8383849263191223, 2.0368399620056152, -0.25046950578689575, 0.14617185294628143, 0.40956538915634155, 1.0580545663833618, 0.5523421764373779, -1.417188048362732, -1.3886263370513916, 0.6834827661514282, 0.17959105968475342, -0.5776238441467285, -0.5513659119606018, 1.8219830989837646, 0.911329984664917, 0.3881607949733734, 1.310524821281433, -0.47138839960098267, 1.3344532251358032, 1.2804744243621826, -0.3233838379383087, 1.0957932472229004, 1.9248017072677612, 0.2201194018125534, 0.3433467745780945, 0.23068082332611084, -0.03610941395163536, -0.6228137612342834, -2.369760751724243, -0.2974580228328705, 0.2650846838951111, -0.04944947361946106, -0.3315836489200592, -0.02123877964913845, 0.19332687556743622, 0.023659873753786087, -1.7427855730056763, -0.6862528920173645, 0.37986886501312256, 0.25287482142448425, 0.18179556727409363, -0.21257656812667847, 0.1138780266046524, 0.5668917894363403, -0.3027958273887634, 0.3438667356967926, 0.0012903634924441576, -1.7630313634872437, -0.1611756831407547, 0.5111598372459412, 0.2663487493991852, 1.0170636177062988, -1.380198359489441, 1.9893064498901367, -0.398534893989563, 1.250179409980774, -0.051914941519498825, 1.0957858562469482, 1.2637951374053955, -1.0578755140304565]], [[0.29379600286483765, -1.033761978149414, -0.022227633744478226, -0.41065865755081177, 0.6247479915618896, 1.1730774641036987, 0.4070069491863251, 2.183236598968506, -2.0736401081085205, -1.7485724687576294, 1.053964614868164, -0.8917016983032227, 0.4117186665534973, -0.8081931471824646, 0.1117062121629715, 0.36029383540153503, 0.5990800857543945, 0.28330445289611816, 2.2143821716308594, 2.1803410053253174, 1.1990606784820557, -2.1607701778411865, 0.22140955924987793, -0.034899987280368805, 0.14492018520832062, -0.6146076321601868, -0.2681445777416229, -1.9361324310302734, -0.2170219123363495, 0.1938774138689041, -0.8273966908454895, -0.1053445115685463, -1.592289686203003, 0.3870091736316681, -1.0036029815673828, 1.3617433309555054, -0.27784961462020874, -0.5684112906455994, 0.1380929946899414, -2.0236308574676514, -0.6723936796188354, -0.7017638683319092, 0.049650639295578, -1.0552089214324951, 1.3163049221038818, 0.08345731347799301, -0.01773170940577984, 0.6760778427124023, 1.6333268880844116, 0.9626531600952148, 1.098420262336731, 0.34171268343925476, -0.7755922079086304, 0.5383521914482117, -0.42015329003334045, 2.191377878189087, 0.0734185054898262, 0.5857579112052917, -1.1671421527862549, -0.32434937357902527, -2.261139154434204, 0.9100593328475952, 0.9073675870895386, 0.593767523765564], [1.5170814990997314, 1.5977137088775635, 1.1097825765609741, -0.23484867811203003, -0.8537803888320923, -1.5090727806091309, 0.060097791254520416, 1.4916484355926514, 0.600660502910614, -0.7815203666687012, -0.953286349773407, -1.1806412935256958, -1.3635003566741943, -1.9252345561981201, -2.101003885269165, 1.1429970264434814, 0.3558933436870575, -0.7574734091758728, 1.68100106716156, 1.4260720014572144, -0.4309532344341278, -1.3317514657974243, 0.20737788081169128, 0.2512137293815613, -0.9999034404754639, -0.5317587852478027, -0.8643555045127869, 1.2553000450134277, -0.10588293522596359, -1.244813084602356, 1.4771668910980225, -1.7421272993087769, 1.7733409404754639, -0.4615406394004822, 0.4720073342323303, -1.6159886121749878, 0.44952473044395447, -0.6486560106277466, -0.5026246905326843, -1.0521682500839233, -0.667905330657959, -0.12482386827468872, 0.38485175371170044, -1.0149205923080444, -0.7133323550224304, 0.8703773021697998, -1.7488943338394165, -0.6375049948692322, 0.953517735004425, 0.876326322555542, -0.22299864888191223, 1.300658106803894, -0.4120556116104126, 1.4720295667648315, 1.5717620849609375, -0.2936106026172638, -0.5298004150390625, -0.5232541561126709, -0.14514151215553284, -1.2038625478744507, 1.119807243347168, -0.7558993101119995, 0.4889379143714905, -1.524030327796936], [1.0423215627670288, 0.983596682548523, 1.612760066986084, 1.10123872756958, -0.6013534069061279, -1.2527459859848022, -1.0850189924240112, -0.3459410071372986, 0.0011796827893704176, 0.1718604862689972, -0.1849510222673416, 0.6654621362686157, 0.1347908228635788, 1.0559043884277344, 0.27782198786735535, -0.3478948771953583, -0.1766221523284912, 0.34102681279182434, 2.04890513420105, 1.0881190299987793, -0.6005089282989502, -0.8889688849449158, -0.8531172871589661, 0.9921748638153076, -1.294685959815979, 0.2756793797016144, -0.6255636215209961, -1.0952508449554443, -0.8821650147438049, -1.15292489528656, 0.49058687686920166, 0.7649242877960205, -2.100938558578491, -1.0556819438934326, -0.4032718539237976, -0.6197871565818787, 0.42998576164245605, -2.2840335369110107, -0.15767435729503632, 1.9707567691802979, -0.4176119565963745, 1.323085069656372, 1.1116623878479004, 0.4004004895687103, -1.0328075885772705, 0.15414002537727356, -0.3025931417942047, -2.4576430320739746, -1.052164077758789, -2.403778076171875, -1.921543836593628, -1.4094626903533936, 0.12357370555400848, 0.6071210503578186, 1.4021791219711304, -0.37923505902290344, 0.41802316904067993, 0.1711309552192688, -0.9933724403381348, -0.4886634647846222, 0.9818835258483887, -0.11560112982988358, -1.2094978094100952, -0.13872216641902924], [0.43047893047332764, -0.009995434433221817, 0.08350736647844315, -1.350221872329712, 0.39654362201690674, 0.1795332282781601, -0.4788714349269867, -1.0603113174438477, 2.2457540035247803, 2.7238283157348633, -0.7984868288040161, 0.3638075292110443, -0.658098042011261, -1.0311447381973267, 1.84538733959198, -1.2139956951141357, -0.4410371780395508, 2.029355049133301, -2.0961828231811523, 0.23334887623786926, 0.08675098419189453, 0.17490459978580475, -0.2966662347316742, -0.4971347749233246, -1.5795313119888306, -0.10364177078008652, -0.13366220891475677, 0.0733254998922348, 1.2002677917480469, -0.7830204963684082, 1.6844456195831299, 0.5677164196968079, -1.053180456161499, -0.7851506471633911, -0.7301799654960632, 0.536493718624115, -1.3137798309326172, -0.5718153715133667, -0.9210494160652161, 0.30074524879455566, -0.06194451078772545, -0.7175102829933167, -0.0889088585972786, 0.7949313521385193, -1.0996707677841187, -0.014156704768538475, 0.10816165059804916, -1.7474092245101929, -1.0791645050048828, 0.4822736382484436, -2.0128064155578613, 0.9939529895782471, -0.45927339792251587, -0.5751082897186279, 0.11580203473567963, 0.41708487272262573, -0.1476687341928482, 1.5062600374221802, -1.09579598903656, 0.9447417855262756, 0.3843725621700287, 0.6536052227020264, -0.10449638962745667, -0.03678072616457939], [-0.15297791361808777, -0.07231124490499496, 0.6085667014122009, -0.9729022979736328, 0.10013991594314575, -0.013524630106985569, -0.28456738591194153, 1.673620581626892, 0.14097757637500763, 0.80058354139328, -0.31467366218566895, 0.17179220914840698, 0.6114381551742554, 0.6152884364128113, -0.36294662952423096, -0.47064119577407837, 0.327472984790802, -0.3139301538467407, -1.02548348903656, -0.9443891048431396, -1.3890875577926636, 0.13578516244888306, 0.4432820975780487, 0.024587156251072884, -1.2562919855117798, 1.2076282501220703, -0.2640548646450043, 0.46357935667037964, 0.09078040719032288, 0.4081536829471588, -1.0134222507476807, 0.5593250393867493, -0.8187503814697266, -0.12383954226970673, -2.20935320854187, -0.04832884296774864, -0.7726936340332031, -0.04305682331323624, 1.2186870574951172, -0.908859372138977, 0.3516464829444885, 0.4598001539707184, -0.2277732789516449, 0.23840896785259247, 1.307302713394165, 1.8786073923110962, -1.1653553247451782, 0.0601806715130806, 1.4325660467147827, -0.9049167037010193, -0.7524288892745972, -0.8603733777999878, 0.9610488414764404, 0.752849280834198, -1.3058717250823975, -0.9571152329444885, -1.2885433435440063, 0.7807366847991943, -0.05771869793534279, 1.7126489877700806, 0.14574839174747467, -0.48622944951057434, -0.8541532158851624, 0.6963585019111633], [-0.8518236875534058, -0.6066771149635315, 0.717199981212616, 0.12775267660617828, 0.648613452911377, -0.9309742450714111, 1.8066269159317017, 1.2414671182632446, -1.195542812347412, 0.7617122530937195, -0.41428443789482117, 1.5195256471633911, -0.35841095447540283, 0.09266890585422516, 1.016907811164856, 0.6160759329795837, -0.25310999155044556, -1.3537373542785645, -3.026968479156494, -1.6814942359924316, -1.2253981828689575, 0.35566699504852295, 0.7345304489135742, 0.5878185033798218, 1.4356215000152588, -0.4829942584037781, -0.8440787196159363, -0.19754669070243835, 0.30870166420936584, 1.960305094718933, 0.5609182715415955, -1.165739893913269, 0.47198301553726196, 1.1746811866760254, -0.6073431968688965, 1.6317228078842163, 0.579432487487793, 2.2909903526306152, -0.36604803800582886, 0.011152789928019047, 1.2359755039215088, 0.2847987413406372, 1.2958487272262573, 0.8523054718971252, -0.9177040457725525, -0.9821090698242188, -0.4148145616054535, -1.115224003791809, -0.014860978350043297, 1.382399082183838, 0.3404565751552582, -1.6284548044204712, -2.297539710998535, -1.7221118211746216, 1.2373459339141846, 0.7104759812355042, 0.5855606198310852, 0.48615968227386475, -0.42279306054115295, 0.7639967799186707, -0.9934387803077698, -1.138543725013733, 0.7253530621528625, 0.10872791707515717], [-0.3704201281070709, 0.4526660144329071, 0.8112892508506775, 0.6678707599639893, -0.3683590292930603, 0.8009180426597595, -0.32437971234321594, -1.2416044473648071, 0.9664980173110962, -0.19214627146720886, -0.06458255648612976, -1.6147701740264893, 1.1559948921203613, -1.0771301984786987, -2.4343974590301514, -0.6781359314918518, -1.8096189498901367, -0.7759394645690918, 1.4019529819488525, 0.022584708407521248, 1.0843050479888916, -2.6290595531463623, 0.2064155787229538, -0.4309016168117523, 0.10509185492992401, 1.69313645362854, -0.3864600360393524, 0.07704392820596695, 0.7701845765113831, -0.49162524938583374, -1.3190439939498901, -0.09427855908870697, 0.8468177914619446, 0.7716147899627686, 0.36441028118133545, -1.08193838596344, -0.38615116477012634, 0.6287286281585693, 0.37753379344940186, 1.8440531492233276, -2.3213307857513428, -1.7623413801193237, -1.2128126621246338, -2.0520455837249756, 0.32394298911094666, -0.38551947474479675, -1.9082914590835571, -0.35881906747817993, 1.1972168684005737, 0.7701236009597778, -0.8030701875686646, -0.5756613612174988, -0.2631971836090088, 1.615256667137146, 0.20046596229076385, 0.13184067606925964, 1.0046559572219849, -0.030847089365124702, 1.9663515090942383, -0.25924161076545715, -0.6229419112205505, -0.17411866784095764, -1.468316674232483, -0.15605765581130981], [-1.3621492385864258, -0.18736222386360168, 1.592934250831604, -1.0766905546188354, 0.3417845368385315, 0.9328853487968445, 2.3193953037261963, 0.88956218957901, 1.080811619758606, 0.7035508155822754, -1.5790297985076904, 0.7957691550254822, 0.008448212407529354, 0.1021423190832138, 0.7547588348388672, 1.151908278465271, 2.43259859085083, 0.8742949366569519, 2.2845916748046875, 0.609062671661377, 0.4759752154350281, -0.9382205605506897, -0.9599371552467346, 0.46422433853149414, 0.833246111869812, -0.4669516384601593, -0.44256526231765747, -0.5191175937652588, -0.10294277966022491, 0.1013961061835289, 1.6599483489990234, 0.12376844137907028, 1.0350134372711182, 0.23110352456569672, -0.5256046056747437, -0.8272733092308044, -0.7213000655174255, -0.32254672050476074, 0.1536577045917511, 1.323004961013794, -0.03629593551158905, -0.7638883590698242, -1.106042742729187, -0.03014250658452511, -0.07748474925756454, 0.6707508563995361, -0.5491423010826111, 0.769892692565918, 0.43832772970199585, -0.4040118455886841, 0.38158607482910156, -0.751358151435852, 0.7642186284065247, 2.476588010787964, 0.5301169753074646, -1.2011585235595703, 1.6916335821151733, -0.07529112696647644, -2.8393423557281494, -0.06798317283391953, 0.16741269826889038, -0.5092650055885315, 0.15742306411266327, 0.5307413339614868], [-0.45807772874832153, -0.03903108090162277, -0.8601473569869995, 0.8925731182098389, -1.4902796745300293, 0.37867531180381775, -0.5116094946861267, -0.68062424659729, -0.8104346990585327, 0.23719905316829681, 0.08327005058526993, -0.8131858110427856, 1.9111049175262451, 1.5344693660736084, 0.14924375712871552, -0.5267887711524963, 0.5705775022506714, 1.2595170736312866, -0.18269850313663483, -0.07713228464126587, 1.8897123336791992, -1.3383625745773315, 0.6070398688316345, 0.1901693195104599, -1.7348350286483765, -1.5005403757095337, 0.8722686171531677, 0.3752634823322296, -0.02308719791471958, -0.289751261472702, 0.3489673435688019, 0.3391347825527191, -1.054229974746704, -0.3640972673892975, 1.040822982788086, -1.7660281658172607, -1.9305497407913208, 0.21674904227256775, -1.1094002723693848, -0.32547810673713684, 1.8084955215454102, -0.6132339239120483, -0.21445880830287933, 0.7276671528816223, -0.5195280909538269, 0.8665388226509094, 0.3445303738117218, -0.07662250846624374, -0.5968497395515442, -0.10394506901502609, 0.5197848081588745, 0.2996247708797455, -0.5613234043121338, -0.6193979382514954, 0.8766134977340698, 0.5916936993598938, -0.6324896812438965, 0.861477255821228, -2.339038610458374, -0.10808485001325607, -1.3515819311141968, 0.04124658554792404, 0.3566575348377228, -0.17075401544570923], [0.8946271538734436, 0.2589859664440155, 0.11692208796739578, -0.25455936789512634, 0.20502027869224548, -1.076751947402954, -0.44640761613845825, 0.19839708507061005, 0.3694593608379364, -0.7589526176452637, 1.1710267066955566, -1.761054277420044, 1.20622980594635, -0.40388426184654236, -0.2140917032957077, 0.0751444548368454, 0.09640590846538544, 0.07065802812576294, 1.3396223783493042, 0.22820593416690826, 0.03395051509141922, 0.20871229469776154, -0.031055081635713577, 0.00781706441193819, -1.2013823986053467, -0.544745922088623, 1.2695859670639038, -0.3392537534236908, -0.6430249810218811, 0.9171012043952942, -0.893949568271637, -1.3784921169281006, 0.5815402269363403, -0.19045524299144745, -0.13171228766441345, -0.2566499710083008, -0.27093350887298584, -1.2089943885803223, -0.04499737173318863, -1.0916670560836792, 1.5690357685089111, -1.1921157836914062, -1.5630552768707275, -0.18350531160831451, -0.28362300992012024, 0.3912138342857361, -1.7118128538131714, -1.3979778289794922, 1.3190160989761353, 0.22612638771533966, -0.45205414295196533, 1.234631896018982, 0.8114027976989746, 0.480839341878891, -0.34000083804130554, 1.1950576305389404, -1.5514708757400513, -0.40064066648483276, -0.6949754953384399, 1.0347529649734497, 0.3681574761867523, -1.2937431335449219, -0.02123730629682541, 0.8520097136497498], [-1.0737712383270264, -0.8227449059486389, -0.8980425000190735, 0.7000059485435486, 0.7447376251220703, -0.46149250864982605, -0.7782326340675354, 0.6777910590171814, 0.9602519869804382, 1.1946234703063965, 1.1997151374816895, 1.3413188457489014, 0.7035079598426819, 0.6874290108680725, 0.6002752780914307, 0.6833246350288391, 0.8459543585777283, -0.7734867334365845, 0.19039767980575562, -1.346190333366394, -1.1428046226501465, -0.06610018014907837, -0.43003031611442566, 1.4798083305358887, -1.6906641721725464, -1.7805860042572021, -1.1678386926651, 0.7438071966171265, 0.06855249404907227, 0.2381594479084015, -2.4281165599823, 0.19235600531101227, -2.0729334354400635, 0.736314594745636, 0.24030406773090363, 0.22118598222732544, 0.5718978643417358, 1.59869384765625, 1.133544921875, -0.8528141379356384, -0.4131859242916107, 0.1357838362455368, -0.9190393090248108, 0.2609397768974304, -1.3476916551589966, -0.5736337304115295, 0.9335604906082153, 0.2616707682609558, 0.5904509425163269, -1.5200273990631104, -0.3430671989917755, -1.3011000156402588, -0.35712724924087524, -0.27167895436286926, -1.8606181144714355, -0.3201800286769867, 0.1842304766178131, 0.6590456962585449, 0.5478622913360596, 1.0723118782043457, -0.19089193642139435, -1.2666757106781006, 0.6460106372833252, -1.0521548986434937], [0.26333826780319214, 1.4150030612945557, 2.391972541809082, -0.37770766019821167, -0.32568392157554626, -0.6307116746902466, 0.3179144859313965, 0.9789026379585266, -1.85917329788208, -0.2946084439754486, 0.49742355942726135, -2.2170333862304688, -0.8298041820526123, -0.14992497861385345, -0.4829234480857849, 1.3945832252502441, 0.8679708242416382, -0.21159985661506653, 1.1515133380889893, 1.9551022052764893, -0.03402986750006676, 0.19396153092384338, 0.8284330368041992, -1.8878570795059204, -0.6083704829216003, 0.6059432625770569, 0.3486049175262451, -0.7404667139053345, 0.2901788353919983, 0.3069913983345032, -1.4480695724487305, 0.4404769241809845, -1.0608876943588257, -1.5834426879882812, -1.483073353767395, 0.2006600946187973, -1.0514267683029175, -1.004096508026123, -1.5216962099075317, -1.346501111984253, 0.12435447424650192, -0.15315060317516327, -0.7858979105949402, -0.3856762945652008, 0.582298219203949, 1.2228384017944336, 1.8206369876861572, -1.5446562767028809, -0.6605222225189209, -0.07600672543048859, -0.932618260383606, 0.788916826248169, 0.38173791766166687, 0.1472935676574707, -0.8398949503898621, 1.3941789865493774, -0.25351396203041077, -1.1325029134750366, 0.9738249182701111, 0.22940292954444885, 0.12906521558761597, 0.31938233971595764, -1.0289374589920044, 1.5030204057693481], [0.854877233505249, -0.5753759145736694, -0.4162086546421051, -0.2551065683364868, -0.8400205373764038, 0.5501042008399963, -0.7265779972076416, 1.0974619388580322, 0.542427659034729, -1.7414653301239014, 0.952863872051239, -0.12106942385435104, 1.1475260257720947, -2.0924460887908936, -0.053663235157728195, 0.38089755177497864, 0.1414269506931305, 0.6163934469223022, 0.5226278305053711, 1.601729154586792, -0.32502487301826477, 0.4119957983493805, -1.557238221168518, -0.1773250848054886, -0.09302826225757599, 0.8291559219360352, -0.4367676377296448, -0.8811146020889282, 0.766208291053772, -0.02593163773417473, 1.2722718715667725, -0.26193737983703613, -1.327939510345459, -0.11844521015882492, -0.10073154419660568, -0.0541553795337677, 0.11179923266172409, 1.043758511543274, 1.0988571643829346, 0.24759405851364136, 0.2310238629579544, 0.7990807294845581, -0.07761833071708679, -0.8836459517478943, -0.620472252368927, -0.8767330050468445, -1.1371934413909912, -1.4449152946472168, -0.020573463290929794, -1.2133811712265015, -1.139507532119751, -1.6064497232437134, -0.3857325315475464, 0.9327858090400696, 1.2051106691360474, -0.693229615688324, -1.4153796434402466, 0.044169262051582336, 1.4147495031356812, 1.8105323314666748, 0.7959572076797485, 0.32315710186958313, 1.887172818183899, -0.230772003531456], [-0.6164863705635071, 0.1258050799369812, -1.02329683303833, 1.0465123653411865, -1.692522406578064, -2.033257246017456, -1.2646284103393555, 1.3956564664840698, 0.027536028996109962, -0.1643088459968567, -0.4911349415779114, -1.279335856437683, 0.23333293199539185, -0.2703148126602173, -0.39281708002090454, 1.8968439102172852, -1.0760177373886108, 0.24435217678546906, 1.2752745151519775, 0.25280094146728516, 0.9808491468429565, -0.42698419094085693, -1.1659730672836304, -0.20958812534809113, 0.6127137541770935, -2.084510087966919, -1.0329614877700806, -0.317966103553772, -0.6239050626754761, 0.6183537840843201, -0.38537171483039856, 0.972830057144165, 0.022487269714474678, -0.15422126650810242, -1.1317192316055298, 0.8260193467140198, 0.5978918075561523, -0.6525976657867432, 1.2700093984603882, 0.37337496876716614, 0.053180795162916183, 0.060180019587278366, -1.6027874946594238, 1.3190213441848755, 1.2467446327209473, -2.1695072650909424, -0.8307439684867859, 0.16637763381004333, 0.5371795296669006, -0.6317961812019348, 0.43277212977409363, 1.3110865354537964, 0.41952526569366455, -2.2577059268951416, -0.26846975088119507, 0.5106538534164429, -0.761814534664154, 1.84787917137146, 1.9579678773880005, -1.5360289812088013, -0.1424192488193512, -0.02178564853966236, -0.1627911925315857, 0.26084548234939575], [0.43661314249038696, -0.2414318472146988, 0.14561425149440765, -1.308881163597107, -0.39816734194755554, -0.5462417006492615, -1.2170478105545044, -0.5744243860244751, -1.1998035907745361, -1.083511233329773, 0.08235546946525574, 0.3518218994140625, -0.8028209209442139, 0.3973790407180786, -0.9316569566726685, 1.1817201375961304, 1.371740460395813, 0.19642938673496246, 1.4336248636245728, 0.6106500029563904, 0.2498319000005722, -0.11763354390859604, -0.09181436896324158, -2.257608413696289, -1.2694711685180664, 0.10023458302021027, 0.28564557433128357, 0.41778290271759033, 0.7471086978912354, 0.5741215944290161, -0.558364987373352, 0.3388823866844177, 2.1512410640716553, -1.155625581741333, 1.397856593132019, -1.6322695016860962, -1.332436203956604, -1.2439290285110474, -0.3471486568450928, 0.4241544306278229, -1.1775776147842407, 2.4698407649993896, 1.1551275253295898, 0.837478756904602, -0.5046268105506897, -0.5168555378913879, 1.1213879585266113, -1.223412275314331, 0.4167473316192627, 0.33169975876808167, -1.7414416074752808, -0.4354480803012848, 0.4401790499687195, 2.595454216003418, -1.1938410997390747, -0.00423900643363595, -1.210082769393921, 0.5995676517486572, -0.21577821671962738, 0.23916643857955933, -0.903936505317688, -0.8932706713676453, -0.799383819103241, 1.526442050933838], [0.09243811666965485, -0.7609816789627075, -0.7441598176956177, 0.14719460904598236, -1.1575403213500977, 0.5655040740966797, -0.8203896284103394, -0.827951192855835, -0.28138190507888794, -0.1989290565252304, -0.09261994063854218, -2.3987975120544434, 0.7135748863220215, 0.6528681516647339, 1.0301791429519653, 0.06899756193161011, 1.273113489151001, -0.11116757988929749, 0.4501156806945801, 1.1567904949188232, 0.6899372935295105, -1.0120898485183716, 2.7503862380981445, -0.345682829618454, 1.336751103401184, 0.2587529420852661, 0.7755630612373352, 1.5761024951934814, 0.621917724609375, 1.1660763025283813, -0.15926624834537506, -1.3559144735336304, 0.7560948729515076, 0.488135427236557, -1.8066319227218628, 0.3141170144081116, 0.6811394691467285, -0.05871397256851196, -0.1572834551334381, -0.6664783954620361, -1.9913420677185059, -0.5943469405174255, -0.5728914737701416, 0.5575925707817078, 0.2384113222360611, -1.367928385734558, 0.10521198064088821, -0.14420832693576813, 2.109382390975952, -1.0449988842010498, 0.18158189952373505, -0.6404210329055786, 0.31317463517189026, -0.4768085479736328, 0.6170493960380554, 0.9749298095703125, -0.23569826781749725, 0.36477240920066833, -0.047832876443862915, -0.06988424807786942, 0.604013204574585, 1.5207693576812744, -1.1347365379333496, 0.08961929380893707]], [[-0.8108217716217041, -0.5062617063522339, -1.0191096067428589, 0.59858638048172, -0.4157624840736389, -0.6956205368041992, 1.396060585975647, -1.0650922060012817, 0.21211354434490204, -0.7994386553764343, 0.7341753244400024, 0.4526841938495636, 0.38891711831092834, -1.0876144170761108, 1.4239203929901123, -0.4551355838775635, -1.352705478668213, -0.6776034832000732, -0.16494300961494446, -0.1512230932712555, 0.08915002644062042, 0.7560480237007141, -1.5829291343688965, 1.616010069847107, -0.011021710000932217, 1.1062259674072266, -0.07007952779531479, -1.7896252870559692, 1.8437044620513916, -1.5055036544799805, -0.89080810546875, 0.5754106044769287, -0.10043148696422577, 1.0737128257751465, -1.350972056388855, -1.184163212776184, 1.9360729455947876, -2.0015106201171875, 0.9859004616737366, -0.8949152827262878, -0.45217764377593994, -1.8409160375595093, -0.19167517125606537, 1.7310082912445068, -0.3193243741989136, 0.21251854300498962, 0.4368169605731964, 0.9217638969421387, -1.4728364944458008, 0.507264256477356, 0.616753339767456, 0.3888758420944214, -0.25431886315345764, -0.48181280493736267, 0.7579705715179443, -0.8228191137313843, 1.3423587083816528, -0.6673862338066101, -1.1833293437957764, 0.7355154752731323, -1.25493586063385, -0.6617661118507385, -0.13775944709777832, 1.6465412378311157], [1.1086344718933105, -0.2845178544521332, -0.3986859619617462, 0.6232139468193054, -1.3754031658172607, 0.03349268063902855, -0.4000994861125946, -0.053989894688129425, -0.7085622549057007, -0.12360750883817673, -0.9853197932243347, 0.8417776823043823, 1.3485510349273682, -1.431314468383789, 1.2043448686599731, 0.73621666431427, 1.1114400625228882, 0.5671207308769226, -1.7491868734359741, 0.8423221707344055, -0.5486358404159546, 0.14014016091823578, 0.22438858449459076, 0.7205467224121094, 0.5544707775115967, -0.4356054663658142, -0.0576472133398056, 1.2470625638961792, 1.1676708459854126, -1.2839834690093994, 0.09018780291080475, -1.0105431079864502, -1.022587776184082, 0.331484854221344, -0.7385722994804382, 0.22200126945972443, -0.8096156120300293, 0.2533736824989319, -0.5065194368362427, -1.4563326835632324, -0.7167063355445862, -1.7625041007995605, 0.08424802869558334, -0.8493521213531494, -1.571010708808899, -1.0613067150115967, 0.34622669219970703, -0.8297327160835266, -0.9065073728561401, 1.4550541639328003, 0.2673538625240326, -0.7574788331985474, 0.42398038506507874, 0.7026776075363159, 1.5293742418289185, -0.8264188170433044, -0.813681960105896, 0.3269205093383789, -1.3046878576278687, -0.48533517122268677, -0.13662704825401306, 0.06345263123512268, 0.17131257057189941, -0.34133824706077576], [0.9721643328666687, -0.6435630321502686, 0.6199789643287659, -1.6274977922439575, 1.0149072408676147, 0.8372604846954346, -1.8610605001449585, 0.5495816469192505, -0.8884925246238708, -0.21403588354587555, -0.804940402507782, -1.4797781705856323, -0.428843230009079, -1.3692868947982788, 0.48183736205101013, 0.5053368210792542, -0.590229332447052, 0.1544562727212906, 0.31811705231666565, -0.2567838132381439, -0.6960275173187256, 0.735867440700531, -0.84342360496521, -0.5542234182357788, 1.2679697275161743, 1.0353416204452515, 0.18649360537528992, 1.091156005859375, 1.6059225797653198, -0.4139987528324127, -1.801291584968567, -1.2492954730987549, -0.10723190009593964, -0.015550727024674416, -0.2737917900085449, -0.03227326646447182, -0.5588713884353638, 0.46886777877807617, 1.6281543970108032, -0.09358493983745575, -1.0045937299728394, 1.3088067770004272, 0.32764577865600586, 1.7854644060134888, 0.6224036812782288, -0.2167828530073166, 1.3093630075454712, -0.6276587247848511, -0.24879446625709534, 0.3533645570278168, 0.8978461027145386, 0.24126039445400238, 0.3311309516429901, 0.23221491277217865, 1.1817430257797241, -1.410203218460083, -0.21040961146354675, -2.100414752960205, 1.6477969884872437, 0.2634263038635254, 0.6352719068527222, 0.6500139236450195, -0.3793676793575287, 0.9674933552742004], [-1.1185302734375, 0.7619010806083679, -0.4845370352268219, 0.9541173577308655, 0.2357150912284851, 0.6733672618865967, -0.7703367471694946, 0.9386944770812988, -0.854851245880127, -0.30533111095428467, 0.5723596811294556, -2.7687931060791016, -1.2814419269561768, 1.63728666305542, -0.11295377463102341, -1.5150829553604126, -0.09186920523643494, -1.2359378337860107, -1.1091368198394775, -0.8008785247802734, -0.34197157621383667, -0.45730727910995483, 0.5741875171661377, 1.0225508213043213, -0.5599552392959595, -1.093846082687378, -0.6004961133003235, -0.34509825706481934, -1.3132585287094116, 0.5741591453552246, 0.8411993384361267, 0.3125031590461731, 0.3710804879665375, 0.19067712128162384, 0.41628867387771606, 0.8165492415428162, -0.8855357766151428, 1.4841724634170532, 0.33906781673431396, 0.414714515209198, 0.4036620557308197, 0.8839324712753296, -0.9824337959289551, 0.8768293261528015, 1.2248480319976807, 0.39880621433258057, 0.25157779455184937, -0.6131690144538879, 1.0392584800720215, 0.7007963061332703, -1.0319058895111084, 0.7059895396232605, -1.304284930229187, -1.4575823545455933, -0.018608370795845985, 0.8541086912155151, -0.8423318266868591, -2.786341667175293, 1.0846461057662964, 0.19264040887355804, 0.6939143538475037, -0.2830562889575958, 1.4822059869766235, -0.09847814589738846], [-0.6087254881858826, -0.09137792885303497, -1.0100418329238892, -0.38932499289512634, 0.21737711131572723, 1.958367109298706, -0.5376480221748352, -0.12458726018667221, 0.48774734139442444, 0.23015885055065155, -1.0408315658569336, 2.674163818359375, 0.20042039453983307, -0.2256462723016739, -1.2940661907196045, -1.1815496683120728, -0.6397846341133118, 0.2885441482067108, 1.418820858001709, 2.315915107727051, 0.5627363920211792, 0.12119091302156448, -0.7284409999847412, 2.344334840774536, -1.9800766706466675, 0.5416402816772461, -0.8678253889083862, 0.5947590470314026, -0.5099436044692993, 1.2499473094940186, 1.9902608394622803, -2.903489828109741, -0.1864517331123352, -0.048054344952106476, -1.973419189453125, -1.1759026050567627, 3.353095293045044, -1.1276881694793701, 0.4941856265068054, -0.6724860668182373, -0.9970352053642273, -1.4313839673995972, 1.4527788162231445, 0.4913719892501831, -0.2348436415195465, -0.5298936367034912, -2.748934030532837, -1.3890035152435303, -1.7417536973953247, -0.052121520042419434, -0.1596732884645462, -0.2985828220844269, 0.5797184705734253, 0.07622235268354416, 0.021560601890087128, -1.14341402053833, -0.90522700548172, -0.5982983708381653, 0.12528346478939056, -2.391662836074829, -0.6270266175270081, 0.3778892457485199, -0.3081560730934143, -0.7684164643287659], [-0.6480324268341064, 0.14340229332447052, 0.5827521085739136, 0.4499548375606537, -0.2740565538406372, -1.1242120265960693, 1.383880615234375, -0.11668703705072403, 0.8061861395835876, -2.4570014476776123, -1.0363898277282715, -1.2391822338104248, -0.18538959324359894, -0.8871958255767822, 1.464768886566162, -0.4467197060585022, 0.23148462176322937, -0.8096698522567749, 1.590395212173462, -0.5243474245071411, 1.122497797012329, -0.5516127347946167, 0.41438159346580505, -0.635617196559906, -1.0521037578582764, 0.6004607081413269, 0.16561995446681976, 1.2286452054977417, -0.3186356723308563, -0.9575469493865967, 0.6135205626487732, 1.729921817779541, 0.39283907413482666, 0.7785207629203796, 1.7607648372650146, 0.9529813528060913, -0.19015395641326904, -0.400452584028244, 0.37136444449424744, -0.3160744309425354, -0.6501811742782593, 0.38876232504844666, 0.48783302307128906, 1.4051870107650757, -1.0159709453582764, 0.7794016599655151, -0.5804306268692017, 0.6144244074821472, -0.2817080318927765, 1.1749849319458008, -0.43217435479164124, -0.4831508696079254, -0.9244054555892944, -1.1304317712783813, -1.7134371995925903, 0.7816197276115417, 1.5412397384643555, -0.9526974558830261, -1.017469882965088, -0.19211457669734955, 1.8124632835388184, -0.8436775803565979, 2.388305187225342, 0.9559189677238464], [-1.1842514276504517, 0.10136216878890991, -1.9799911975860596, 0.3560991585254669, 0.8275438547134399, -0.6547729969024658, 0.30031704902648926, 0.5976992249488831, -0.9831905364990234, -0.08753586560487747, -0.2945746183395386, -1.749563455581665, 0.22413575649261475, 0.3996920883655548, 0.7545836567878723, -0.12686507403850555, 0.16610322892665863, -0.09753858298063278, -0.8736509084701538, 0.6878985166549683, 0.7266064286231995, -0.7901099324226379, -1.0999011993408203, 1.5736931562423706, 0.3323396146297455, -0.5207479596138, -0.15501351654529572, -0.48719900846481323, 0.5512398481369019, -0.7832298278808594, 1.0113649368286133, 1.2540888786315918, -1.4310388565063477, -0.2760842740535736, -0.49102792143821716, -0.08046353608369827, 0.055476605892181396, 0.6373344659805298, -0.4502585530281067, 0.14656361937522888, 0.5339588522911072, 0.46038818359375, -1.0991755723953247, -0.31760865449905396, 0.9191721081733704, 0.43303579092025757, -1.1865882873535156, 1.5717754364013672, -0.06773082166910172, 0.49580445885658264, -0.0810810849070549, 1.2671180963516235, 0.9382593631744385, 0.3124609589576721, 0.15127642452716827, 0.5498105883598328, 0.5616100430488586, 0.7367844581604004, -0.9171274900436401, -1.3158997297286987, 0.10808456689119339, 0.4097357988357544, -0.8293839693069458, -0.5659608244895935], [-1.6037253141403198, -0.8259798288345337, 1.310821771621704, -0.826941192150116, 0.6040141582489014, -0.40524429082870483, 0.2597787082195282, -0.8576651811599731, -1.2342439889907837, 0.3181736469268799, -0.4543704390525818, 0.013740234076976776, -1.6280326843261719, 0.5742838978767395, 0.28023040294647217, -0.42335838079452515, -0.9883494973182678, -1.1675443649291992, 1.4210543632507324, -0.46860963106155396, -0.46819302439689636, 0.11694495379924774, -1.613903284072876, 0.6496901512145996, -0.3553488850593567, -0.3957936763763428, -0.43632572889328003, 0.20334628224372864, -0.05946258082985878, -0.6851460337638855, -0.1556297093629837, -0.8703758120536804, 1.3356060981750488, -0.5914511680603027, -0.43153539299964905, 0.5324031710624695, 1.1065006256103516, 0.4634275436401367, 0.4286671578884125, -1.2501957416534424, -1.2474853992462158, 0.10427656024694443, -2.092494249343872, -0.02886008657515049, -0.1412764936685562, -0.7792001366615295, -1.5680887699127197, 0.2567473351955414, 1.9466561079025269, -0.3231861889362335, -1.0205491781234741, 0.8446351289749146, 1.435383915901184, 0.8953104615211487, -1.56322181224823, -0.6558473110198975, 0.7088415026664734, 1.9035191535949707, -0.9363176226615906, 1.2015407085418701, 0.8421400189399719, -0.9926269054412842, 0.29781574010849, -0.9132325053215027], [-0.12686608731746674, 1.6987214088439941, -0.22526657581329346, 0.6320297122001648, 1.0237222909927368, 1.0412917137145996, 0.22462831437587738, -0.08174294978380203, 0.9721484184265137, -0.5429718494415283, 0.48603397607803345, -1.4667561054229736, 0.06857725232839584, -0.382470041513443, 0.31633689999580383, -0.5830503702163696, -0.08900438994169235, 1.2956290245056152, -0.9924084544181824, -0.38895225524902344, 0.07091198116540909, -2.5187294483184814, 0.2273178994655609, 1.717881441116333, 0.19095386564731598, -0.047827672213315964, -2.147381544113159, -0.9804943203926086, 0.06645741313695908, -0.5024636387825012, -1.2268580198287964, -0.46935394406318665, -0.06403493881225586, -1.6687862873077393, -1.273267388343811, -0.7613643407821655, 1.0705313682556152, -0.6813364624977112, -2.561659812927246, -2.8396525382995605, -3.3065364360809326, -0.23663851618766785, -0.5373290181159973, 1.7519769668579102, 0.1109187975525856, -0.39982739090919495, 1.0091158151626587, -0.10340479761362076, 0.048912566155195236, -0.8502042889595032, 0.06826183199882507, -0.5092670321464539, 0.6710600256919861, -1.2668678760528564, 0.3941337466239929, 0.18251748383045197, -0.8148084282875061, -1.966932773590088, -0.7623695135116577, -0.5820465087890625, -0.6685307025909424, -0.8748776316642761, 0.9843993186950684, -1.015629529953003], [0.5403697490692139, 1.7715132236480713, 1.3404792547225952, 0.6260108947753906, -0.565959095954895, 1.1941218376159668, 0.5288747549057007, 0.8729874491691589, -0.038458529859781265, -0.15122991800308228, -0.3218701183795929, 0.6495691537857056, 0.4360849857330322, -1.1165674924850464, -0.7047882676124573, -0.11000417172908783, -0.05542590469121933, 0.4497741162776947, -0.11568254977464676, 0.1332564800977707, 2.06733775138855, 1.6627885103225708, -1.2368907928466797, -0.36926743388175964, 0.514597475528717, 0.47980910539627075, -2.0229427814483643, 0.21975252032279968, 1.7734614610671997, 0.8768730759620667, -0.07300413399934769, -1.1835501194000244, 0.4953381419181824, -0.40962302684783936, 0.19756382703781128, 0.501523494720459, 0.3480397164821625, 0.3653925955295563, 0.043569695204496384, -1.2995288372039795, -0.042209066450595856, -0.15591368079185486, -0.06503307074308395, -0.6612363457679749, -1.4183028936386108, 1.126502275466919, 0.3242468237876892, 0.6553371548652649, 0.04047120362520218, 0.14413122832775116, -0.02477244846522808, 0.2572459578514099, -0.854138970375061, 0.3430667817592621, -1.196600317955017, -0.6552523970603943, -0.7731720805168152, 1.5246881246566772, -1.0476070642471313, -0.24573221802711487, 1.4857852458953857, 0.060995347797870636, 1.6705434322357178, 0.7932440638542175], [-0.8761656880378723, 1.434401512145996, 0.7470182776451111, -1.2284992933273315, -1.9238088130950928, 0.8795962333679199, -0.976458728313446, -0.11215785890817642, 0.1313599795103073, 0.8297985196113586, -0.260751873254776, -1.2971296310424805, -1.0822908878326416, -0.07420941442251205, 0.8923530578613281, -0.5604590773582458, -1.1927109956741333, -1.7129440307617188, 1.831176996231079, -0.8945080637931824, -0.8180778622627258, -0.4469364583492279, 0.5830597281455994, 0.3479999303817749, -1.3774000406265259, -1.9189939498901367, -0.7222215533256531, -0.05845959112048149, -0.7012341618537903, 0.3483732044696808, -0.5442785620689392, 1.7226295471191406, -0.5497955083847046, -0.7877218127250671, 1.0027556419372559, 0.11624602973461151, -1.4857133626937866, -0.5350291132926941, 0.03638904169201851, -3.158050060272217, -1.058684229850769, 0.45977839827537537, 0.5582239627838135, 0.14955785870552063, -1.4337059259414673, -0.3900126814842224, -0.9962124824523926, -0.19722333550453186, -0.42111000418663025, 0.4703396260738373, -1.2974553108215332, -0.6769660115242004, 0.026296434924006462, 0.544815719127655, -1.352258563041687, -0.1882728487253189, 2.3508832454681396, -0.02998480387032032, -1.2426518201828003, 0.8667811155319214, 0.4152630865573883, 0.15602776408195496, -0.13315492868423462, 0.7979995012283325], [-1.1120878458023071, 1.0234758853912354, 0.8432161211967468, 0.9551160335540771, 0.7853630781173706, 0.8530339002609253, 0.06845501065254211, -0.3855501115322113, 0.2677370607852936, -0.648896336555481, 1.297553539276123, 1.6835461854934692, -1.2748066186904907, 0.7495880722999573, 1.7296981811523438, -1.1880521774291992, 1.0081596374511719, -0.3490184247493744, 0.9228730797767639, -1.1342352628707886, 0.48851102590560913, -1.442508339881897, 0.9037883281707764, 2.0066051483154297, -0.6801832318305969, -0.38057902455329895, -0.49868831038475037, -0.17262284457683563, -0.683193027973175, -1.297165036201477, 0.19644126296043396, -0.13660310208797455, -0.3935316801071167, -1.6235889196395874, -0.45297664403915405, 0.3673171103000641, 1.8695319890975952, 1.178902506828308, -2.3433215618133545, 0.7012152671813965, -0.10217411071062088, -0.020414341241121292, -0.8455340266227722, 0.19163087010383606, 0.14699847996234894, -0.1956997662782669, -2.022486686706543, 0.3505612313747406, -1.0416916608810425, -0.11160045117139816, -0.04914337024092674, -1.6019082069396973, -0.08691605925559998, -1.3505754470825195, -0.01697418838739395, -1.133294701576233, 0.00045579299330711365, -0.5610684156417847, 1.1101741790771484, -0.7011329531669617, -0.1697668433189392, -1.3695828914642334, -0.4972992539405823, -0.4174233078956604], [-2.1533427238464355, 0.2780894637107849, 0.9293559193611145, 1.3983900547027588, 0.6385074257850647, -0.39436987042427063, -0.5610167980194092, -0.15230222046375275, 0.7444414496421814, 2.2802646160125732, 1.127907156944275, 0.825374186038971, 1.3582744598388672, -0.06785822659730911, 0.4491623640060425, 0.6151761412620544, 0.10047239810228348, -2.2148709297180176, -1.2812169790267944, -0.2120654433965683, 0.06590209156274796, 0.09917894750833511, -0.0850449651479721, 1.230357050895691, 0.3859768211841583, -0.8417347073554993, 0.08405528962612152, -0.6689485311508179, -0.25106489658355713, 1.8801192045211792, 0.946444571018219, -0.596778929233551, 1.0175116062164307, 2.5795109272003174, -0.41062262654304504, -0.672478199005127, 0.8153509497642517, -0.9929235577583313, -1.177337646484375, -0.247749462723732, -2.275952100753784, -1.2247092723846436, -0.4233790934085846, -0.44109007716178894, -0.5768764019012451, 2.1136474609375, -1.7541754245758057, 0.6883673667907715, 0.1823730319738388, 0.5209910869598389, -1.3642973899841309, -0.11465055495500565, -0.29235997796058655, 0.937405526638031, -0.43567267060279846, -0.9311327934265137, -0.3519708216190338, 0.4003846347332001, 0.3131067454814911, 0.802956223487854, 0.702176034450531, 0.614444375038147, 0.979072630405426, 1.7529891729354858], [0.6656903624534607, 0.025498678907752037, 0.2606123685836792, -0.24471785128116608, 0.9466665387153625, 0.16219455003738403, 1.6588948965072632, -0.42755892872810364, 1.5932048559188843, 1.3428199291229248, -0.2257310003042221, -0.09820124506950378, -0.2141498625278473, 0.10055015236139297, 0.36284077167510986, -1.0083260536193848, 0.777600884437561, -0.3705707788467407, 0.21185439825057983, -0.08160558342933655, 1.4065001010894775, 0.004135641735047102, -0.8304876089096069, 1.259500503540039, -0.12625224888324738, -0.0349152572453022, 0.3104855716228485, -0.26645445823669434, 0.4018051028251648, -0.879584014415741, -0.9232315421104431, 1.6896276473999023, 0.8986268043518066, 0.15712012350559235, -1.2849053144454956, -0.5697464346885681, 0.9877853393554688, -1.1755801439285278, -0.8608309030532837, -0.21503467857837677, -0.12041272968053818, 1.7380025386810303, -0.040064696222543716, 0.4080647826194763, 0.5259643197059631, 0.19861535727977753, -0.8528923392295837, 0.6070464253425598, -1.2807505130767822, -1.0176126956939697, -0.603259265422821, 0.9015291333198547, -0.30807051062583923, -1.9597833156585693, 1.4398270845413208, 0.5164029002189636, 0.28486624360084534, 0.7694216966629028, -1.0336265563964844, -1.3554480075836182, 0.5565475821495056, -0.11413849145174026, 0.06618990749120712, -1.4211928844451904], [-0.11838334053754807, 0.9249824285507202, -1.030845284461975, -0.4469893276691437, 0.20604687929153442, 0.37310898303985596, 0.36888599395751953, 1.8132249116897583, 0.6609694361686707, 1.131730079650879, -0.4450690746307373, -0.11175280064344406, -1.4361861944198608, 0.5117630362510681, 1.1012203693389893, -0.9735636711120605, 1.8503384590148926, 0.202415332198143, -0.34419843554496765, -1.5976817607879639, -1.0718026161193848, -0.06990957260131836, -1.032163143157959, -0.6320728063583374, 0.11301866173744202, -0.9421790242195129, -0.6846460700035095, 0.11562883108854294, -1.5112251043319702, -2.0922369956970215, 1.1970323324203491, -0.8123003244400024, 0.3279111385345459, -0.9558683037757874, -0.17359785735607147, 1.2204068899154663, 0.5559923648834229, -1.2449339628219604, -1.0444884300231934, -0.5058140754699707, -1.0600193738937378, 0.036862414330244064, -1.1629917621612549, -0.9148513078689575, 1.2269625663757324, 0.478664368391037, -1.6032990217208862, 0.9571318626403809, -1.0785127878189087, 0.3971026539802551, 0.1932820826768875, -1.174519419670105, 0.727136492729187, -0.9564926028251648, 0.005550199653953314, 0.017953073605895042, -0.8708004951477051, -0.8198776245117188, -0.7947651147842407, -1.2349377870559692, 0.9023256301879883, 0.3733726441860199, -0.034994032233953476, 0.20747129619121552], [0.8758167624473572, -0.5873808860778809, -0.2132566124200821, 1.3492509126663208, -0.22415225207805634, 0.3430132567882538, -1.0141472816467285, 0.019463999196887016, 2.427140235900879, -1.0315823554992676, 0.0424790121614933, 1.3243080377578735, 1.1465797424316406, -0.46590936183929443, 1.726773977279663, 1.3697088956832886, -2.842111587524414, 0.9116547703742981, 1.5876805782318115, -0.6129937171936035, -0.42704978585243225, -0.8255356550216675, 0.735486626625061, 1.6204748153686523, 1.131628155708313, -0.03551413491368294, 2.7839114665985107, 0.45410189032554626, -1.350529670715332, -2.0960347652435303, 0.36125314235687256, 0.8437719941139221, 0.2853107452392578, -0.36072418093681335, -0.7897381782531738, -0.1353083997964859, -0.8219181299209595, 1.710304856300354, -0.7322357892990112, 0.6007552742958069, 0.5921450853347778, 0.7786808609962463, 0.6499759554862976, 0.9687625765800476, 0.8810689449310303, 0.06656274199485779, 1.5387448072433472, 1.5022687911987305, -0.660043478012085, -0.14395633339881897, -0.6788117289543152, -0.46183621883392334, -0.13534463942050934, 0.46900472044944763, -0.1339959353208542, -0.42349618673324585, -0.5369873046875, 0.8339105844497681, 0.602751612663269, -0.11639517545700073, 1.340225100517273, -0.1537684202194214, -0.11151652038097382, 0.009145679883658886]], [[1.254486322402954, 0.09234260022640228, -0.23623865842819214, 0.0022732627112418413, 1.89565908908844, -0.4733649790287018, 0.6138757467269897, -1.0313355922698975, -0.7504520416259766, 0.40530768036842346, 0.19281764328479767, -0.6141998767852783, -0.6645469069480896, -1.425936222076416, 0.5992810726165771, -0.6684064269065857, -1.4814926385879517, 0.9466201663017273, 0.49561673402786255, -0.15506060421466827, -0.9554294347763062, -0.7528620958328247, 1.527987003326416, 0.2353033572435379, -0.3253372311592102, 1.055663824081421, 1.47284734249115, -0.5754345059394836, 0.3777865171432495, -1.2737423181533813, -0.009960454888641834, -0.5545715689659119, -0.8826829195022583, -0.5261927247047424, 0.6532803177833557, -0.9481467604637146, -0.008594760671257973, -0.27969396114349365, -0.5765721797943115, -1.9334310293197632, -0.20899051427841187, -0.5055789947509766, 0.3021266758441925, 1.068616509437561, -0.4239632487297058, -0.1900772750377655, -0.1137954443693161, -0.008864933624863625, -0.418031245470047, 2.640353202819824, 1.6757533550262451, -0.8910405039787292, -0.8331870436668396, 1.5326248407363892, -0.9930164813995361, 0.3353513777256012, -1.8990983963012695, 1.895397424697876, -0.1892758160829544, 0.5618941783905029, -0.4052532911300659, 1.082837700843811, -0.38296130299568176, 1.3361860513687134], [1.051093578338623, 0.3400823175907135, 0.29405975341796875, 0.8602290749549866, 0.9941594004631042, -1.4064619541168213, 0.8040024638175964, -0.49689966440200806, -0.1018182560801506, -0.28630563616752625, 1.6546876430511475, 0.9666355848312378, -0.8947629332542419, -0.6771166920661926, 0.6950398683547974, 1.6066093444824219, -0.9614517688751221, -1.8675062656402588, 0.43080976605415344, -1.573994517326355, 0.3559541702270508, 0.6806436777114868, 1.010437250137329, -1.1801533699035645, 0.9609536528587341, 2.303304433822632, 1.3759400844573975, 0.5768452286720276, 0.19439971446990967, -2.9273335933685303, -0.33838868141174316, -0.556199312210083, 0.7343938946723938, -1.045401692390442, 2.350724220275879, 0.9320806264877319, 1.5602083206176758, 1.586914300918579, 2.8894875049591064, -1.1677616834640503, 0.5944746732711792, -0.506796658039093, -0.010494403541088104, 0.01685679331421852, -1.2710251808166504, 0.6837683320045471, 1.4069912433624268, -1.2845102548599243, 0.10437534749507904, 0.7366681694984436, -0.0881725400686264, 0.8360627889633179, 0.4748457074165344, 1.0667074918746948, -0.2399580329656601, 0.5680872201919556, -0.0597250834107399, 1.262528896331787, 2.0253729820251465, -0.698793888092041, -0.18513137102127075, -0.042568061500787735, 0.20413580536842346, 1.375461220741272], [-0.08011199533939362, -0.6390484571456909, -1.2734531164169312, 0.19904173910617828, 1.5093164443969727, -0.22149065136909485, -1.5829826593399048, 0.026317445561289787, -0.04026658460497856, 0.24408580362796783, 0.20365332067012787, 2.1303939819335938, -0.3344924747943878, 0.20206992328166962, -0.40246084332466125, -2.1725685596466064, -0.9953146576881409, -0.7031156420707703, 0.12412043660879135, -0.09429235756397247, -0.5726944804191589, 0.18957899510860443, -0.9796783924102783, -1.220031976699829, 0.48600271344184875, -1.0998215675354004, 0.7450482845306396, 0.28250938653945923, -1.7386428117752075, 0.04159202054142952, 1.640862226486206, 0.5279664993286133, -0.6000640988349915, 0.8453179597854614, -0.8179137110710144, -2.022470712661743, -1.4800978899002075, 1.1471139192581177, 1.390639305114746, -0.9529756903648376, -0.5501164197921753, -0.8563632369041443, 1.7011433839797974, -0.4013345241546631, 1.056357741355896, -0.7855393886566162, 0.16394451260566711, -1.4057338237762451, -1.6731528043746948, -0.861010730266571, 0.7589048147201538, -1.0416964292526245, -0.3087104856967926, 1.1154849529266357, 0.4760136008262634, 0.8846256136894226, 0.8713136911392212, 0.9426550269126892, -0.42536500096321106, 1.6586655378341675, -0.7063673138618469, 0.6656580567359924, -0.3989678621292114, 1.3697690963745117], [-0.05133689567446709, 0.4599374532699585, -0.7738202810287476, 0.4507061243057251, -0.4334682524204254, 0.4907001852989197, -0.4825705289840698, 0.40518853068351746, -1.0063841342926025, -1.9737944602966309, 1.3013256788253784, 1.2324798107147217, -1.1296066045761108, 1.0009828805923462, 1.3801240921020508, -0.6957630515098572, 0.1588834524154663, 0.2669041156768799, 0.3122367560863495, -0.18114538490772247, 0.35541632771492004, 1.3557676076889038, 1.9633965492248535, -1.639173150062561, -1.3289260864257812, -0.08373536169528961, -0.6738297939300537, 0.4782455861568451, -0.6913794875144958, 2.2787723541259766, -0.24938742816448212, -0.19702908396720886, -0.1454777866601944, 0.6471844911575317, -0.042814306914806366, 0.36853039264678955, 0.5364194512367249, -0.04788120836019516, 1.0303760766983032, -0.5617873072624207, 0.6837433576583862, -0.5400437116622925, -0.9546551704406738, 0.7485347986221313, -0.14855964481830597, 0.4369436502456665, 2.366891622543335, 0.18670734763145447, -0.732357382774353, 0.7006641030311584, -0.7018889784812927, 0.044227030128240585, -2.6940081119537354, 0.9231756329536438, -1.3635894060134888, -1.013353705406189, 0.3768293261528015, -0.5636170506477356, 0.5570676326751709, -0.9837700128555298, 0.6596919894218445, 0.5487020611763, -0.5188086032867432, 1.0505313873291016], [1.7328282594680786, -0.8591100573539734, 0.16382086277008057, 1.02608060836792, -0.857750654220581, 0.01695091463625431, 0.9175348281860352, -0.36336854100227356, -0.373641699552536, 1.519412636756897, -0.20587623119354248, -0.6461209654808044, 0.30445072054862976, 2.021820545196533, 0.05109196528792381, 0.5199925303459167, 0.3594219982624054, -0.7722760438919067, -0.5767354369163513, 1.1106635332107544, 0.29588884115219116, 1.0441148281097412, -0.01896689273416996, -0.701978862285614, 1.6045583486557007, 1.681281328201294, 0.4296543002128601, -0.34889936447143555, 0.41567757725715637, -0.08320286870002747, 0.5600736737251282, -0.06555716693401337, -0.9219725131988525, -0.44945451617240906, 1.4181963205337524, 0.07600300759077072, -0.7953028678894043, -0.09073822945356369, 0.41024935245513916, 0.38550329208374023, 0.8241957426071167, 0.9747191071510315, 0.33275020122528076, 0.30090534687042236, -0.7184060215950012, -1.8277779817581177, 1.2284889221191406, -0.6062048673629761, -0.10514110326766968, -0.32993951439857483, 0.24823863804340363, -1.8963385820388794, -0.4035358726978302, -0.6818127036094666, -1.2908868789672852, -0.19044053554534912, 0.5790769457817078, -0.5030381083488464, -0.8667823076248169, 0.06504832208156586, 1.3898332118988037, 1.5390851497650146, 0.2902604043483734, -0.29333144426345825], [0.3969423174858093, -0.2297351062297821, 0.6674098372459412, -0.2296467125415802, -1.004698395729065, 1.3327476978302002, -0.6955792903900146, -0.2973088026046753, -0.1445763260126114, -0.1359063982963562, -0.6372644305229187, 1.155012845993042, 0.5856202244758606, -1.7717849016189575, 0.44061699509620667, 1.310434103012085, 0.9296697974205017, -0.013109183870255947, -0.9374477863311768, 0.17329655587673187, -0.618979275226593, 0.7090063691139221, 1.7014459371566772, -2.729278326034546, 0.6168829798698425, 0.8529706001281738, 0.10805223882198334, -0.5566285848617554, 0.309123694896698, -0.8407822847366333, 2.2328009605407715, 1.4967283010482788, -2.1728620529174805, -0.6143198609352112, 1.5849697589874268, 1.4876859188079834, -1.074368953704834, 1.3596889972686768, 0.6220659017562866, -0.002222247887402773, 1.887699007987976, 0.4388011395931244, -0.3217565417289734, -1.22614324092865, -0.9415585994720459, 0.5252453088760376, -0.11975947767496109, 0.5567493438720703, 0.879229724407196, -1.181449055671692, 0.42171695828437805, 0.31977540254592896, 1.1751965284347534, 0.21884162724018097, 0.8756730556488037, 0.18733927607536316, -2.1001925468444824, 0.5433297157287598, 1.6686519384384155, 0.4932098090648651, -0.6612712740898132, 0.7601012587547302, -1.3917354345321655, -1.424584984779358], [0.5900898575782776, -0.036063481122255325, -0.08774916082620621, 0.7565192580223083, -0.8146538734436035, 1.7768582105636597, 2.0284385681152344, -0.09019559621810913, -0.2586174011230469, -0.7357426285743713, 0.5027526617050171, 1.0485484600067139, -0.36406758427619934, -0.016205357387661934, -1.8689731359481812, 0.45781585574150085, 0.3931941092014313, 0.009473816491663456, -0.2040414661169052, 2.7073147296905518, -1.320896029472351, 0.5991062521934509, 0.5439258217811584, -0.5045467019081116, -0.6319723129272461, 1.281265377998352, -1.2828916311264038, 0.36223000288009644, -1.035904049873352, -0.38049882650375366, -0.8599603176116943, 1.5454280376434326, -2.6143643856048584, -0.6392390727996826, 1.3356754779815674, -1.1579034328460693, -0.0012400925625115633, 1.0903840065002441, 2.1889283657073975, 0.5332349538803101, -0.7501007914543152, -0.3563743531703949, -1.8078855276107788, -0.4164092242717743, -0.6006200313568115, 0.5730782747268677, 0.034618597477674484, -1.8098491430282593, -0.35296958684921265, 1.0967013835906982, -0.3509405553340912, 0.9433785676956177, -0.12023957073688507, 0.27587321400642395, 0.30662181973457336, -0.10602748394012451, -1.6579396724700928, 1.2300201654434204, -0.3705892264842987, -0.3439942002296448, -0.3494061827659607, 0.25394973158836365, 0.5740320682525635, -0.9738729000091553], [-0.6617287397384644, 1.419137716293335, -1.6583671569824219, 0.757900595664978, -0.4410153031349182, -0.4485146105289459, 1.192224383354187, -1.3696157932281494, -0.1555447280406952, 1.9712462425231934, -0.6800957322120667, -0.7842507362365723, -0.6022161841392517, 2.0734875202178955, 1.1394822597503662, 1.267562985420227, -0.5406359434127808, 0.3467550575733185, 0.20780010521411896, 0.08632142841815948, -0.6551579833030701, -1.0006450414657593, 0.43547049164772034, 0.4006590247154236, -0.8212209343910217, -0.43081313371658325, 0.6496454477310181, 1.1129027605056763, -1.5023505687713623, -1.298144817352295, 0.1290665715932846, -1.2241308689117432, -0.6136072278022766, -0.324585497379303, 0.23025673627853394, -2.3550639152526855, -1.2292860746383667, 1.3266342878341675, 0.6628735661506653, 0.31042271852493286, -0.2140570729970932, 0.17298947274684906, 0.6411919593811035, -1.0679529905319214, 0.7342174053192139, -0.11358863115310669, -0.01682031899690628, 1.373594880104065, -0.12456347048282623, 0.38982337713241577, -0.8585120439529419, -1.797844648361206, 0.3549498915672302, -0.08086742460727692, 1.5454070568084717, -0.020721156150102615, -0.08078524470329285, 0.6450209021568298, 1.0976808071136475, 1.4000024795532227, -0.2051495462656021, -0.7326139807701111, -0.5148844718933105, 1.164844036102295], [-1.1035879850387573, -0.8977574706077576, 0.07443231344223022, -0.8403390049934387, 0.1543159782886505, -0.05557269975543022, 2.406235933303833, -1.5643484592437744, -0.44164684414863586, 0.9369006156921387, -1.0498065948486328, 1.4818105697631836, -0.6984427571296692, -1.1291102170944214, 0.4727080464363098, 0.3051779568195343, 0.38344278931617737, -1.4008910655975342, 0.46076104044914246, -0.6872954964637756, -1.5473839044570923, 0.14238132536411285, 1.0986188650131226, 0.6259728670120239, -2.030421495437622, 1.0342715978622437, -0.6445484161376953, 0.08141064643859863, -0.5733956694602966, -0.9434992671012878, 0.21986964344978333, -0.19142337143421173, -1.4195626974105835, 1.0663944482803345, 2.1819093227386475, -0.325400710105896, -0.051696132868528366, 1.1284198760986328, 0.18715178966522217, 0.3754727840423584, -1.7014411687850952, -1.6795580387115479, -1.332910180091858, 0.10510756820440292, 1.2279325723648071, -0.27337104082107544, 0.134952113032341, 1.1781647205352783, 0.4312902092933655, 0.04474771022796631, -0.07339678704738617, 0.989196240901947, 0.2005060613155365, 1.1094529628753662, -0.6407033205032349, -0.732649028301239, 0.7427920699119568, 0.5483769178390503, -0.03122003935277462, -0.16030998528003693, 0.2227596491575241, -0.13946785032749176, -1.349852204322815, 1.2738895416259766], [1.5178894996643066, 0.8526248335838318, -0.02024882286787033, -0.028470490127801895, 0.8561342358589172, -0.005369804799556732, 1.5824239253997803, 0.18311017751693726, 0.2934928834438324, -0.4799100458621979, -1.766545295715332, 1.1870999336242676, -1.1552674770355225, 0.5170052647590637, 1.9760956764221191, 0.3467336595058441, -0.21423275768756866, -0.9902697801589966, 0.1854947805404663, -1.0880377292633057, 0.20408016443252563, -0.425953209400177, -0.29888153076171875, 1.0295583009719849, -0.06896030157804489, -0.32088226079940796, -0.4115464389324188, -0.30475643277168274, 0.6725583672523499, -0.5479241609573364, -0.746456503868103, 1.3547990322113037, 1.3744926452636719, -0.05150971561670303, 0.5477932095527649, 0.5533096194267273, 0.017471060156822205, 0.2966715097427368, -0.19525626301765442, -0.8190592527389526, -0.09622941166162491, -0.02955729141831398, 0.4041266143321991, 0.008385873399674892, -0.119330994784832, 0.02926531806588173, -1.062907338142395, 0.8145274519920349, -0.1350114643573761, 0.1935250610113144, -0.8593724370002747, 1.9583882093429565, -1.6604657173156738, 0.5690098404884338, -0.6615419387817383, 0.01357161346822977, -0.7689329385757446, 0.1389205902814865, -1.4909082651138306, -1.8161593675613403, 0.01803554594516754, 0.4097355902194977, 0.6786646842956543, 0.5613609552383423], [-0.2842416763305664, -0.2757139205932617, 0.16297060251235962, -1.2657184600830078, 1.42116117477417, -0.10733456909656525, -1.3671163320541382, 1.8563836812973022, 0.42229294776916504, 0.8371570110321045, -0.0348573736846447, 1.077755331993103, -0.03605236858129501, 0.24263033270835876, 0.5682504177093506, -0.7212643623352051, -0.5867546200752258, 1.5793696641921997, 0.673146665096283, -0.18436671793460846, -0.1443823277950287, 1.6477750539779663, -1.161731243133545, 2.5764379501342773, 0.8581480979919434, 0.6012915968894958, -0.08143725246191025, -0.7542558908462524, 1.7173258066177368, 0.569124162197113, 0.2389427125453949, -0.4374189078807831, -0.2546921670436859, 0.005248097702860832, 0.10576923936605453, 0.45317336916923523, -0.7456457614898682, -1.161926507949829, 0.9156950116157532, 1.1355931758880615, -0.4564942419528961, 1.1495084762573242, 0.11332248151302338, -0.7149174809455872, 1.879555344581604, 1.97102689743042, 0.5867334008216858, 0.06134738400578499, -0.10014092177152634, 0.784268319606781, -0.5227826237678528, -0.4481256604194641, -1.1917757987976074, -0.22230233252048492, 1.1099984645843506, -0.15226411819458008, -0.858089804649353, 0.014218173921108246, 0.9385407567024231, -0.9361422061920166, 0.7839502096176147, -1.0716632604599, -1.0214333534240723, 0.08049675077199936], [1.930928111076355, -0.28655993938446045, -1.074705719947815, 0.026180265471339226, 1.6166929006576538, 0.8723781704902649, -0.5454716086387634, -0.7357169389724731, 0.22672700881958008, -0.3166390359401703, 0.43341290950775146, -0.4340493977069855, 0.5577810406684875, -0.36785560846328735, -1.137155532836914, 0.1837969720363617, 1.0158584117889404, 0.08294340968132019, -1.1734592914581299, -0.4040977656841278, 1.6384060382843018, -1.2882870435714722, 0.26256704330444336, -1.8330214023590088, -0.9077407717704773, 0.6912578344345093, 0.44079697132110596, 0.0626671239733696, -0.14830847084522247, -0.24312925338745117, -0.8641943335533142, -0.028921185061335564, 0.3239539563655853, 0.21150928735733032, -1.1148195266723633, 0.9734564423561096, -1.3247804641723633, 0.7705498337745667, -0.36596807837486267, -0.3651595711708069, -1.9348198175430298, 0.03342161327600479, 0.9929704666137695, 0.04216454178094864, -0.49434277415275574, -0.5137221217155457, -0.3308502435684204, -1.0935019254684448, 0.5900043845176697, 0.3376641273498535, 1.7697583436965942, 0.8968605399131775, -0.832683801651001, -1.4468683004379272, 1.4414862394332886, 0.2673836350440979, -0.4308736026287079, -1.1064413785934448, 0.9582619071006775, 1.5154838562011719, 0.546137809753418, -1.3205689191818237, 0.9438039660453796, 0.07939749956130981], [-1.2554677724838257, -0.2714936435222626, 0.23767568171024323, -0.3950667977333069, -1.9573999643325806, -0.4982661008834839, -0.8151910901069641, 0.8241630792617798, -0.3092364966869354, -2.267634630203247, -0.7605307102203369, 0.9292632341384888, 0.5152273774147034, -1.2669649124145508, -0.9888591170310974, -0.6695317625999451, 1.3451124429702759, 0.3685375452041626, -1.2309516668319702, -0.2308957725763321, -1.11933434009552, 0.10492845624685287, -0.3450464606285095, -0.7203315496444702, 0.7949134707450867, -0.27062538266181946, -1.7617638111114502, 0.6949855089187622, -0.8708959221839905, -0.7491686940193176, 0.1101294755935669, -1.101638913154602, 0.05905924364924431, -2.5742650032043457, -0.9221000671386719, 2.0456416606903076, 0.2703128457069397, 0.4960068166255951, -0.16666658222675323, -0.06744701415300369, -0.9222181439399719, -0.1688355952501297, -1.9821383953094482, 0.5716105699539185, -0.38511282205581665, 0.5018463730812073, 0.02891436032950878, -0.5977847576141357, -0.13469235599040985, -1.2661432027816772, 0.7218348979949951, 0.6053847670555115, 0.1387947052717209, -1.297383427619934, 1.0112591981887817, -0.7945533990859985, -0.32754021883010864, -0.5278482437133789, 0.12232554703950882, 0.2144654095172882, -0.7176479697227478, 1.4139032363891602, -1.1421078443527222, 0.7400149703025818], [-0.49746766686439514, -0.6187182068824768, 0.30169764161109924, 0.5091976523399353, 0.7509012818336487, -0.9584770798683167, -0.29538488388061523, 0.3994344472885132, -0.24135783314704895, -0.33086490631103516, 0.6990493535995483, 0.7554682493209839, 0.15268419682979584, -1.3364534378051758, -1.5728411674499512, -0.8130767941474915, 0.405691921710968, 0.5439302921295166, 2.95871901512146, 1.0739291906356812, -0.027142774313688278, -0.4459717869758606, -0.3778790533542633, 0.7117545008659363, -0.23226284980773926, -0.22598621249198914, -0.12651561200618744, 0.7133755683898926, -0.3240633010864258, -1.841986894607544, 1.7038300037384033, -1.5193142890930176, 0.570756733417511, 0.8149915337562561, 1.0883798599243164, -0.16047236323356628, -0.0819181427359581, 0.14829590916633606, -0.03670230135321617, -0.3303065001964569, -0.6352334022521973, 1.9373103380203247, -0.9124931693077087, -1.926808476448059, -2.032707929611206, -0.09354919195175171, -0.546625018119812, -2.1128151416778564, -1.237405776977539, -0.7983731627464294, -1.3533908128738403, -0.5610562562942505, 0.5268997550010681, 1.6116293668746948, -0.7418019771575928, -0.1959889978170395, -0.5806713700294495, -0.2986430823802948, 1.8360776901245117, -1.0807591676712036, 1.6639589071273804, 0.03620420768857002, -0.2325177937746048, 0.629131019115448], [0.39671790599823, -1.1543651819229126, 0.10580408573150635, -0.4617915153503418, 1.966755986213684, -0.9983991384506226, -0.2449445128440857, 0.9999015927314758, 0.990925133228302, 1.6138345003128052, 0.2734874188899994, 2.659619092941284, 0.3170609176158905, 1.2267723083496094, 0.6903663873672485, -1.2727234363555908, -0.5474052429199219, -1.1593492031097412, -0.1788429468870163, 0.9574762582778931, 0.2341923713684082, 0.6849752068519592, -0.40806183218955994, 0.4765303134918213, -0.7336244583129883, 0.6966680884361267, -0.8019495606422424, 0.2671279013156891, 1.2109185457229614, -1.277207612991333, -0.18860094249248505, 0.26447793841362, -0.6861206293106079, -0.5145655274391174, -0.589881420135498, -0.8915826678276062, -0.41016626358032227, -1.9141225814819336, 0.5848650932312012, -0.7720810174942017, 0.4000387191772461, 0.678636372089386, -0.8738877177238464, -0.973773717880249, 0.6089136600494385, -0.4085315465927124, 0.21050405502319336, 0.19894099235534668, 1.9623091220855713, -0.8330149054527283, 0.1304449439048767, -0.3889632523059845, 1.3562932014465332, -2.1128182411193848, -1.6504998207092285, -0.1330500692129135, -0.5233599543571472, 0.8940584063529968, 1.9150326251983643, 0.027773914858698845, 0.318155974149704, -0.9195266962051392, -1.6093964576721191, 0.40098655223846436], [-0.2415427416563034, -2.223515272140503, 1.106835126876831, -0.03128212317824364, -2.15484619140625, 0.36139190196990967, 0.8427788615226746, 1.1205402612686157, -1.7581310272216797, -1.8194336891174316, 0.19573993980884552, -0.8311980962753296, 0.6516581177711487, 0.5526671409606934, -0.759917140007019, 0.9250849485397339, 1.1609731912612915, 0.2512975335121155, -0.49470487236976624, -1.2595027685165405, 0.4171736240386963, -0.061851102858781815, 0.731107234954834, 0.0946880429983139, -1.395862102508545, 0.6913979649543762, -0.030489269644021988, -0.603939950466156, 0.8961422443389893, 0.44589146971702576, -0.570766031742096, 0.49133995175361633, -0.39051303267478943, 0.6860380172729492, -1.214501142501831, 1.2420191764831543, -1.4271035194396973, 0.04663907736539841, -1.2433795928955078, 0.0020327649544924498, -0.8485859036445618, -0.5230616331100464, -0.8734955787658691, 0.18050476908683777, -0.4274454116821289, -0.8812066912651062, -1.3141154050827026, 1.5758639574050903, 0.6991616487503052, 1.0660849809646606, 0.835947573184967, -0.7094964385032654, 1.3937515020370483, -2.533212423324585, -0.35179856419563293, 0.26363393664360046, -0.06514538079500198, -2.4594321250915527, -1.1713896989822388, 0.4506359398365021, 0.39905601739883423, -2.036639451980591, 1.209232211112976, -0.7133708596229553]]]], "dtype": "float32"}, {"__nd__": [[[[-0.7392309308052063, 1.0613659620285034, -1.8418433666229248, -0.5448198318481445, -0.8449132442474365, 1.5745280981063843, 0.7666898965835571, 1.1683930158615112, -0.9695287346839905, 0.6420978307723999, -1.1271424293518066, 1.9708565473556519, -0.8387977480888367, -0.8470180630683899, -0.1867648810148239, -0.7236369252204895, 0.4150848388671875, -2.594101905822754, -1.4291977882385254, -0.49087169766426086, 0.607330322265625, -1.8257704973220825, 1.9168661832809448, -0.9942984580993652, 0.9511772394180298, 1.1189039945602417, 1.0379713773727417, -1.8082610368728638, -0.09366113692522049, 0.7770522236824036, -1.3662649393081665, -0.6099796891212463, 0.3429250121116638, 0.035387855023145676, -0.301724374294281, 0.2977226674556732, -1.6424857378005981, 0.49570050835609436, 0.2616593539714813, -1.4863725900650024, 1.4190727472305298, 0.4760321378707886, -1.540210485458374, 0.45527854561805725, -0.4815206229686737, 1.239449381828308, 0.37637120485305786, 0.9681996703147888, 1.2040401697158813, -1.233989953994751, 0.0832611545920372, 1.0115677118301392, 1.5188175439834595, 1.0827977657318115, 0.20136523246765137, 0.2714795768260956, 1.0787492990493774, -0.3876892030239105, 1.4943758249282837, -0.007025897968560457, -1.1131606101989746, -0.6851240396499634, 1.3054429292678833, -0.0714552104473114]], [[0.3983820378780365, -1.3772249221801758, 2.465254068374634, -0.28950127959251404, -0.1404246687889099, -0.3630053400993347, -0.1709207445383072, -0.041876327246427536, 0.3634560704231262, -0.4820344150066376, -1.228871464729309, -0.44429469108581543, -1.2454125881195068, 0.6777396202087402, -1.2521214485168457, -0.2776854336261749, -0.45383980870246887, -1.130686640739441, 0.8865922689437866, 0.2696767747402191, -0.5439797639846802, 0.8442064523696899, 0.2793535888195038, -0.1931031048297882, 0.1057659238576889, -0.8760663866996765, 0.7355189323425293, -0.12479784339666367, -1.1539658308029175, 1.5380021333694458, -0.49970757961273193, -1.4923601150512695, -0.6619400978088379, -0.16763611137866974, 1.7743306159973145, -0.2124159336090088, -0.1899750828742981, -1.1723887920379639, -0.6436547040939331, -0.45482534170150757, -0.5261010527610779, 0.6517183780670166, 0.3937457203865051, 0.9239131212234497, 0.05636700242757797, 2.458667278289795, 0.5549972653388977, 1.2331023216247559, 0.0352318175137043, -1.6085805892944336, -0.6082191467285156, 0.7990738749504089, -0.6824512481689453, 1.1453282833099365, 0.958700954914093, -0.29311999678611755, -0.10845762491226196, 1.1025612354278564, -0.6072331070899963, -0.20503880083560944, 1.6333868503570557, -0.3453291058540344, -0.12001066654920578, -0.5380480885505676]], [[-1.256474494934082, 0.07736048102378845, -0.6027345061302185, -0.9821063876152039, 0.9222771525382996, -0.6194740533828735, 1.0727521181106567, -0.09689264744520187, 0.5969699025154114, 0.32795432209968567, -0.4161047339439392, 0.0652676597237587, 0.6484112739562988, -1.1996281147003174, 0.6564721465110779, -1.2158125638961792, -0.008612111210823059, -1.0739315748214722, 0.9835004806518555, 0.21167947351932526, 0.2746770977973938, 1.756014347076416, -1.2474831342697144, -0.05324898287653923, 1.448454737663269, -1.5204936265945435, 0.9716125726699829, 0.8052693605422974, 1.0382273197174072, -0.8313061594963074, -0.07689738273620605, -1.7105482816696167, 0.003470249241217971, -1.5629953145980835, -0.6972870826721191, 0.5603207349777222, -0.4006485641002655, -1.760683536529541, -0.019914474338293076, 2.3259084224700928, 0.3398369252681732, 0.6846471428871155, -0.3849954605102539, 0.7178567051887512, 0.3988131582736969, 0.2513168156147003, 0.24449530243873596, -0.03218535706400871, -0.231904074549675, 0.017980191856622696, -0.9276303648948669, 0.249210387468338, 0.12176751345396042, 1.5521655082702637, -0.8411056995391846, -1.8283753395080566, 2.134498357772827, -0.1205352246761322, 0.09174773097038269, -1.1921781301498413, 0.3241455852985382, 0.05617665871977806, 0.08089356124401093, -1.4574646949768066]], [[0.2376844435930252, -1.0341466665267944, -1.1783154010772705, -1.376650094985962, -0.3012644946575165, 0.1899993121623993, 0.39792871475219727, 1.2573450803756714, -0.6109111905097961, 1.263148546218872, -0.20879870653152466, -1.2961629629135132, 0.5655114650726318, 1.6571152210235596, -0.3604848086833954, 1.5028095245361328, -0.13337182998657227, -0.08298732340335846, -0.7100608944892883, -1.8428410291671753, -0.19885297119617462, -0.1772790402173996, 2.2174551486968994, -1.4922338724136353, -0.5297401547431946, 0.2872321605682373, -0.2231827676296234, -1.1552789211273193, -1.3329817056655884, 1.1977821588516235, -1.2902368307113647, -0.09040362387895584, -0.5368956923484802, 0.1963765174150467, -0.029086623340845108, 0.6171804070472717, 0.1898716241121292, 3.531663179397583, -0.5755002498626709, -0.6307498216629028, 1.1526025533676147, 1.5878263711929321, 0.7764789462089539, -0.8319397568702698, 2.6008217334747314, -0.42470046877861023, -1.1448540687561035, 0.6872486472129822, -0.01886928081512451, 0.5388840436935425, -1.2328318357467651, 0.838860273361206, 0.12417730689048767, -1.250189185142517, -0.7904442548751831, 0.07810982316732407, -0.5305485725402832, 0.14061017334461212, -0.21894745528697968, -1.4027373790740967, -1.0650700330734253, -0.34562447667121887, 0.10337165743112564, -1.4452776908874512]]], [[[-0.30486148595809937, -1.4221748113632202, -0.8767101168632507, -0.6313964128494263, 0.2837396562099457, 1.0913490056991577, 1.4976435899734497, -1.872179627418518, -0.0200730599462986, 0.6831691265106201, -1.4714542627334595, 1.5139306783676147, -1.6283832788467407, -0.523383378982544, -0.4690706431865692, 0.9417255520820618, -0.4545987844467163, -0.32279106974601746, 0.560083270072937, -0.40471139550209045, 0.8774431347846985, -0.8093648552894592, 1.105682611465454, 0.05309370160102844, -0.3286221921443939, 0.4068012535572052, 0.3482084274291992, 0.188704714179039, 0.7564824819564819, -0.13857129216194153, -2.298637866973877, 0.5332653522491455, 0.52315354347229, -1.4307637214660645, -1.1740083694458008, -0.022053195163607597, 0.0750250518321991, 0.7971007823944092, 1.8140162229537964, 2.144070625305176, -0.4092775583267212, -0.30946987867355347, -0.3459314703941345, 0.27465569972991943, 2.0760715007781982, -0.07451585680246353, -0.745926022529602, -0.28910887241363525, -0.18030573427677155, -0.5830947756767273, -1.0252373218536377, -0.8923237919807434, 0.8154400587081909, -1.573602557182312, 1.4361863136291504, 2.1328213214874268, -0.7587401866912842, -0.32067596912384033, -0.08545047789812088, 1.3760613203048706, 0.7047609090805054, 1.0328459739685059, -1.2356033325195312, -0.743843138217926]], [[0.6571003794670105, 0.9516076445579529, -0.7081255912780762, 0.6946509480476379, -0.11901203542947769, -0.45571425557136536, 0.14059719443321228, -0.22661854326725006, -0.213186115026474, -0.4733521044254303, -0.8519110679626465, 3.143766403198242, -0.43265828490257263, -1.5934560298919678, -0.25186729431152344, -0.8862674832344055, 0.27560314536094666, 0.6952568888664246, -0.31996434926986694, 0.7123983502388, -1.1307231187820435, 1.9201085567474365, 0.4347623586654663, 2.023414373397827, -1.6430987119674683, 1.6876749992370605, -0.603007435798645, 0.4601212739944458, 0.11504985392093658, -2.3331620693206787, -0.6533132195472717, -1.1302424669265747, 1.2718693017959595, -1.4425193071365356, -0.0745634138584137, 2.509397268295288, 0.11017163097858429, -2.757603168487549, 0.2488696575164795, 0.01762229949235916, -0.5459121465682983, -1.0588841438293457, -0.5349489450454712, -0.7041156888008118, 1.1483572721481323, 0.7383132576942444, -0.595747709274292, -0.7563234567642212, -0.758194625377655, -0.10435797274112701, -0.8048838376998901, -1.11711847782135, 1.132141351699829, -0.232492133975029, 0.6751916408538818, -2.0760695934295654, 1.9282574653625488, 1.7921422719955444, 0.2754351794719696, -2.5560898780822754, 0.8070176839828491, -0.6118294596672058, 0.550127387046814, 0.21763576567173004]], [[-0.290200799703598, -1.392781138420105, 0.5661929845809937, -0.6166269183158875, 0.0653839036822319, -0.7319149971008301, 0.0123514449223876, -1.2240556478500366, 1.8576774597167969, -0.08712191134691238, 0.36802250146865845, 0.5836265683174133, -0.599924623966217, -0.9084541201591492, -0.5517690181732178, -1.2804356813430786, -0.4839371144771576, -0.13571538031101227, 1.0018854141235352, 1.4212778806686401, -1.4169940948486328, 0.3891146183013916, 0.563980758190155, 0.42916014790534973, 0.850999653339386, 1.3724244832992554, 0.5843043327331543, -0.5991141200065613, -0.23706476390361786, -0.3004901111125946, 1.7593046426773071, -0.49212607741355896, -0.8056045174598694, 0.5157224535942078, 1.1607221364974976, 1.5162609815597534, -0.2659449577331543, -1.212491512298584, 0.194652259349823, 0.8241933584213257, 0.5178263187408447, 1.681849718093872, 0.5093337297439575, 1.019230842590332, 0.8537811040878296, 0.009187937714159489, -0.6005809307098389, -0.9390529990196228, -1.5198556184768677, -0.5399970412254333, -0.7366099953651428, -1.5985201597213745, 0.17771707475185394, 0.17498652637004852, -1.8447539806365967, -0.4632081687450409, -0.9064469337463379, 0.26871350407600403, -0.24723869562149048, -1.1145049333572388, 0.22349804639816284, -0.6247085928916931, 0.8020216226577759, 0.4552645683288574]], [[-0.663969099521637, 0.7995557188987732, 0.5369426012039185, 0.002145790494978428, -1.137333631515503, -0.8407456278800964, 0.30502432584762573, 0.29731619358062744, -0.9588124752044678, 0.9920663833618164, 0.7141096591949463, -0.19602802395820618, 1.7946892976760864, -0.7484628558158875, -0.035543292760849, -2.7489593029022217, -0.711776852607727, 0.6979305744171143, -0.8893195986747742, -1.3582650423049927, -0.8265739679336548, -0.486960232257843, 2.020353317260742, -0.5797953605651855, -0.3463629186153412, -0.7229843139648438, 1.8777228593826294, 0.8198115229606628, 0.5247331261634827, -0.2493753731250763, -1.3262466192245483, 1.765669345855713, -1.015419363975525, 0.5180742740631104, -0.21859139204025269, 1.4480770826339722, -0.011580993421375751, -0.3051691949367523, -0.581817090511322, -0.9165605306625366, 1.0266815423965454, -0.5303985476493835, 1.0130730867385864, 0.45001816749572754, -0.11824021488428116, -0.4650169312953949, -0.30790233612060547, 0.0350063256919384, -0.0997951477766037, -1.200459361076355, 1.5188223123550415, -0.1661575436592102, 0.6361232995986938, 0.8897782564163208, 1.2180192470550537, 0.5060712099075317, -0.9252456426620483, -0.5926746726036072, -0.6995473504066467, -1.3645051717758179, 0.46575552225112915, -0.2304844856262207, 2.1123509407043457, 0.08435209840536118]]], [[[0.8116713762283325, -2.562244176864624, -2.472961187362671, -2.133685350418091, -1.6024397611618042, -0.7117371559143066, 2.35969614982605, 0.9085426330566406, 0.04270664229989052, -1.2238283157348633, 2.188469648361206, -0.7494634985923767, 0.3501553535461426, -1.1231659650802612, 0.5902402997016907, 1.2873951196670532, 0.7526429295539856, -1.0776621103286743, -0.7210087776184082, 0.9142172336578369, -0.8201493620872498, 0.47661644220352173, -1.0137940645217896, 0.1679125428199768, -2.2980422973632812, 0.6625785827636719, 0.11341046541929245, -0.8602260947227478, 0.3764364421367645, -0.6019319891929626, -0.6090534329414368, -1.9560526609420776, 0.6975065469741821, -1.260946273803711, 1.7201279401779175, -0.22950786352157593, -0.6532708406448364, 0.07754950225353241, -2.085042715072632, -1.830549955368042, -0.024768801406025887, -0.5373451709747314, -1.0525233745574951, 0.6692802906036377, 0.38364726305007935, -1.2398136854171753, -2.253085136413574, 0.9682740569114685, -0.028563646599650383, 0.9532544612884521, -2.098395586013794, 1.6403875350952148, 0.05898125469684601, -0.7054075598716736, 1.1408369541168213, 0.21305902302265167, -1.317258596420288, 2.211751937866211, -2.070161819458008, -0.21281060576438904, 1.093808889389038, 1.2900470495224, 0.4188418388366699, -1.445914387702942]], [[1.0541706085205078, 0.2961692214012146, 0.525797963142395, -1.5518851280212402, 1.4857627153396606, 0.763570249080658, -0.6568708419799805, -0.3033944070339203, -1.8889753818511963, 0.2724611461162567, 2.8427984714508057, 0.3620487153530121, 0.9515774250030518, -1.6018849611282349, -0.8905314803123474, -0.3234977126121521, 1.0859897136688232, -0.31532710790634155, 0.24507199227809906, -0.8003293871879578, -1.078762173652649, -1.137813687324524, 0.7498624920845032, 0.3661361038684845, 1.1572999954223633, 1.037384033203125, -2.3955652713775635, -0.6755806803703308, -0.7903016209602356, 1.2096126079559326, 1.5445072650909424, 1.2291769981384277, -1.8304953575134277, -0.08726323395967484, -0.6446091532707214, 1.1568281650543213, 1.6011215448379517, -0.6751309633255005, 0.08684392273426056, 0.6811370849609375, -0.03007311373949051, -0.23178614675998688, -0.04813679680228233, 1.2830649614334106, -0.40201058983802795, 1.417435884475708, 0.48390260338783264, -0.5863320231437683, -0.3616847097873688, -0.02958529256284237, -0.8357441425323486, 0.4951636493206024, 0.2936406433582306, 0.6578715443611145, -0.2058636099100113, 1.0760610103607178, -0.14610476791858673, 1.1131384372711182, -0.757931649684906, -0.7147996425628662, -0.5921200513839722, 1.7942190170288086, -0.08907552808523178, 0.5700144171714783]], [[2.0084805488586426, 0.8239649534225464, 0.4596250653266907, -1.277248501777649, 0.9399021863937378, -1.0534101724624634, -0.72225022315979, 0.21817976236343384, -0.6072782874107361, 1.7814607620239258, -1.1104204654693604, 0.23513199388980865, 0.28677645325660706, 1.7890305519104004, 0.9724376201629639, 1.1428756713867188, 0.4018353819847107, 1.0142242908477783, 1.5308815240859985, -1.070124864578247, -0.924452006816864, -0.07550151646137238, 1.0186148881912231, 0.13910485804080963, -1.7607356309890747, 0.16388604044914246, -1.397768497467041, 0.06299221515655518, -2.546806573867798, -0.20776477456092834, 0.9298987984657288, 0.3447285294532776, -0.670768141746521, 1.5762829780578613, -0.5481013059616089, -0.08106397092342377, 0.6225740313529968, -0.5099205374717712, 0.3079323172569275, 2.0524306297302246, -0.46192672848701477, -0.6605169773101807, 0.6368751525878906, -1.0939756631851196, -0.21611396968364716, -1.6795365810394287, 0.8811679482460022, -0.5915853381156921, -0.2534053325653076, -1.3825361728668213, -2.244352340698242, -0.2145938277244568, 0.3120184540748596, -1.3462501764297485, -0.15002460777759552, -0.3501906991004944, 1.2764954566955566, -1.2839815616607666, -0.4812203049659729, 1.96257483959198, 0.058710455894470215, -0.5375595688819885, 0.3475242853164673, -0.5571439266204834]], [[-2.2820851802825928, 0.44820863008499146, 1.4160405397415161, -0.022318322211503983, -2.182304859161377, -1.352728247642517, -0.8872042894363403, -0.5304996967315674, -0.3127555251121521, -0.2877546548843384, -0.5370592474937439, 0.029422592371702194, 0.27520033717155457, 0.2474212795495987, 2.195962429046631, -0.20317067205905914, 1.0732600688934326, -1.0423176288604736, 0.5445671677589417, -0.7417594790458679, 0.6199212074279785, 0.5446746945381165, -0.5425122976303101, -1.2720372676849365, 1.5949980020523071, -0.49779820442199707, -0.5080084204673767, 0.039243459701538086, -0.653679609298706, -0.5775099396705627, 1.639513373374939, -0.27606886625289917, -0.3025437593460083, -0.22928868234157562, -0.6552258729934692, -0.492587685585022, -0.9644147157669067, -0.39705419540405273, 0.032705195248126984, 1.4220342636108398, 0.8290674686431885, -0.525603175163269, 0.20035982131958008, 0.3255347013473511, 0.08541785180568695, -0.35189396142959595, 1.0026289224624634, 1.2849174737930298, -1.982796549797058, 0.16865168511867523, -0.15619522333145142, 0.2708267271518707, 1.2579092979431152, 1.8139309883117676, -0.02406446821987629, 0.09537358582019806, -1.3855427503585815, -0.8324429988861084, 0.9775243997573853, -0.46951693296432495, -0.5805636048316956, -1.4031047821044922, -0.9190128445625305, -2.0233800411224365]]]], "dtype": "float32"}, {"__nd__": [[[[1.8236860036849976, -0.4948176443576813, -1.6548939943313599, 0.8525108695030212, 0.23392602801322937, -0.09739526361227036, 1.1184484958648682, 0.6407340168952942, -0.5115337371826172, 0.014273080043494701, 2.3764126300811768, 0.4918953478336334, -0.4286913275718689, 1.2802897691726685, 0.6752033829689026, -0.24169336259365082, 1.0262638330459595, -1.4908397197723389, -0.14501754939556122, 0.768665075302124, -2.1209254264831543, -2.6070568561553955, -0.5639466643333435, -0.37381288409233093, 0.20886926352977753, 0.8504350185394287, -0.41378721594810486, -1.1666158437728882, -1.665786862373352, 0.6232264637947083, 0.6302474141120911, 0.763708770275116, 0.017941638827323914, 0.36980846524238586, -0.6489982604980469, -0.1855204850435257, 1.5768555402755737, 0.6897691488265991, 0.9889079928398132, -0.8455129265785217, 0.8628105521202087, 0.7188106775283813, -0.43376046419143677, 1.6524296998977661, 2.1344106197357178, -0.2881642282009125, -0.8555700778961182, -0.5740684866905212, 0.36849454045295715, -1.5388137102127075, -2.040254592895508, 1.1418769359588623, -1.2641302347183228, -0.5077762603759766, 0.4596583843231201, -0.24508202075958252, 2.36586856842041, 0.34485459327697754, 0.5177354216575623, -1.006761074066162, -0.3329792618751526, 1.065652847290039, 1.187278151512146, 0.10287303477525711]], [[0.06185400113463402, -0.8235948085784912, 0.19253064692020416, 0.2827695608139038, -1.3714927434921265, 0.5311341881752014, 0.86271071434021, 1.5629594326019287, -0.10588192939758301, 0.46681615710258484, -0.16774162650108337, -1.0486611127853394, 1.7207000255584717, -1.1576908826828003, 1.1262727975845337, 1.0041074752807617, 0.5838934183120728, -1.2609498500823975, -1.5902308225631714, -0.7676109671592712, -0.00577115174382925, 1.8010281324386597, -0.6872944831848145, 0.6379814743995667, -1.8714624643325806, 0.48010149598121643, 0.7186427712440491, -0.9268929362297058, 0.15836840867996216, 1.5475445985794067, -0.7555047273635864, 1.011986494064331, 0.6779957413673401, -1.5188812017440796, 0.2688027322292328, -0.7787172794342041, -0.04914805665612221, -0.2758077383041382, -0.3099488914012909, 0.17023533582687378, -0.9076672792434692, -0.98777174949646, -0.6719154119491577, 1.302855372428894, 0.26654958724975586, -1.03343665599823, -0.3674343228340149, 0.4713963270187378, -0.34701162576675415, -0.14605388045310974, 1.4909985065460205, 0.16891898214817047, -0.8413754105567932, -1.1552656888961792, 0.23974063992500305, -0.2868299186229706, -0.558385968208313, -0.9529391527175903, 2.755465030670166, -0.47368136048316956, -2.6118218898773193, 0.5705859661102295, -0.4540863633155823, -0.5117237567901611]], [[-2.2001209259033203, -0.1426621675491333, 0.020197194069623947, 0.3192386329174042, -0.16573050618171692, 0.14354798197746277, 0.48728305101394653, -0.3086296021938324, -1.0261517763137817, -0.3848457634449005, 1.8283039331436157, -1.0740768909454346, -0.8231782913208008, 1.4075512886047363, -1.3145859241485596, 0.28207725286483765, -0.6857382655143738, -0.2710734009742737, 0.14363351464271545, 0.5711228847503662, -0.5377380847930908, -1.600156545639038, 1.447962760925293, -0.7840142250061035, -0.014633748680353165, -0.7832173109054565, 0.49433019757270813, -0.8065822720527649, -0.935127317905426, -0.06134650856256485, 0.7127781510353088, 0.3801880180835724, -0.3817749321460724, 0.19079239666461945, 1.1219202280044556, 0.8697821497917175, -1.0397334098815918, 1.1908445358276367, 0.8479481339454651, 0.34873759746551514, 0.24308332800865173, -0.08370077610015869, 0.41668009757995605, -0.32194405794143677, -1.7117291688919067, 0.10659442842006683, -0.08503958582878113, -1.087611436843872, -0.043415311723947525, -0.3586272895336151, -0.1985875964164734, -0.9727531671524048, 0.876069962978363, 0.49104106426239014, 0.8324479460716248, -0.5226740837097168, 0.5771183967590332, 0.9130811095237732, -1.6782475709915161, 0.15919385850429535, 0.515839159488678, 1.0324840545654297, -0.2529771327972412, -0.2048211693763733]], [[-0.9949422478675842, 0.2459852397441864, 1.4011441469192505, -0.4544806182384491, 1.6836357116699219, 0.43121233582496643, 1.7454942464828491, 1.5476747751235962, -0.09927249699831009, 0.4284362196922302, -0.9569141864776611, -0.4550188183784485, 0.34702858328819275, 0.4070546627044678, -0.20722752809524536, -0.8545247912406921, 2.2298429012298584, 1.334288477897644, 0.5154473185539246, -0.25196269154548645, -0.7519041895866394, 1.0532323122024536, 0.5065150856971741, 1.0464574098587036, 0.18798570334911346, 0.6984654068946838, 1.7496424913406372, -0.3099786639213562, -1.1198660135269165, 0.9405900239944458, 0.5338239073753357, 1.9220249652862549, 1.173625111579895, 1.790632724761963, 0.15807189047336578, 0.8112704157829285, -0.7717803120613098, -2.2176334857940674, -0.15446162223815918, -0.22200097143650055, 0.18319638073444366, -0.5729701519012451, 0.5423130989074707, 0.27359718084335327, 0.46528688073158264, 2.242260694503784, -0.8552306890487671, 4.084046363830566, 0.8815862536430359, 0.10341586917638779, 0.10506941378116608, -0.2846904993057251, 0.485767662525177, -1.649118423461914, 0.7382707595825195, -1.7266148328781128, -0.14403890073299408, 1.3906160593032837, 1.36977219581604, 1.2746858596801758, -0.4066186845302582, -1.3265650272369385, 0.5411103367805481, -0.6360713243484497]]], [[[1.3768837451934814, 0.09882281720638275, 1.2859530448913574, -0.3612605035305023, -1.2282301187515259, 0.9617994427680969, 1.6982532739639282, 0.01001308299601078, -1.1401745080947876, 0.8516960144042969, -1.0769305229187012, -0.2703292965888977, 2.0172910690307617, 1.670407772064209, 0.15178938210010529, 0.3110356032848358, -0.020504871383309364, 0.8337008357048035, 0.9870441555976868, -0.40664443373680115, 0.5729527473449707, 1.3514623641967773, 0.4547847509384155, -0.531658411026001, 1.0399391651153564, 1.380882978439331, 0.10910982638597488, 1.218248963356018, -0.6479478478431702, 0.7668328881263733, 0.7346134781837463, -1.7169047594070435, -0.5094572305679321, 0.5825349688529968, 0.10688353329896927, -0.14163753390312195, -1.6828101873397827, 0.9559517502784729, 0.8023573756217957, -1.7258431911468506, -2.1093387603759766, -0.1386052817106247, -0.10965678840875626, -0.4553765058517456, 0.189395010471344, -1.50806725025177, 0.1764560341835022, -0.6853334903717041, 0.08075812458992004, 0.003398864297196269, 1.1190191507339478, -0.22150374948978424, 0.4025951027870178, 1.770532488822937, 1.4088866710662842, 0.14061643183231354, -0.4684417247772217, -1.500036358833313, 0.23704794049263, -0.940963089466095, -0.9402354955673218, -0.9104337692260742, -0.49971726536750793, -0.5227342844009399]], [[1.1324599981307983, -0.532013475894928, 0.4639238119125366, -0.10458142310380936, 0.10479670017957687, -0.26910319924354553, -1.2524768114089966, -0.6077625751495361, 0.7913779020309448, -0.8178504109382629, 1.1359254121780396, -0.7899373173713684, -1.1559451818466187, 0.9687789082527161, 1.3785392045974731, 0.007240555714815855, 1.332907795906067, 0.6121354699134827, 1.6068378686904907, -1.704201340675354, -1.8911688327789307, 0.24531099200248718, -1.266616702079773, 0.993873119354248, -2.351719379425049, 0.25971633195877075, 0.950161874294281, -1.0229315757751465, -0.16115717589855194, -0.07413479685783386, -0.5152167677879333, 0.3810807466506958, 0.29930418729782104, -0.9799110889434814, -0.14835035800933838, -0.5864117741584778, -2.9209070205688477, 1.2903249263763428, 0.887884795665741, -1.034501552581787, 2.972668170928955, -0.9715632200241089, 0.8049347400665283, -1.1144750118255615, -0.4561734199523926, 0.5645124316215515, 0.2672058343887329, -2.084841012954712, 0.29822903871536255, 1.9902853965759277, 0.755501389503479, 0.9066084623336792, -0.6369521021842957, 0.03870975971221924, -1.6198474168777466, 0.5027350783348083, -1.4993168115615845, 1.4653387069702148, -0.40709465742111206, 1.7880405187606812, -1.2349756956100464, -0.3652817904949188, -0.038364339619874954, -1.2031856775283813]], [[1.8772408962249756, -0.3607428967952728, 1.1871836185455322, -1.0809824466705322, 0.849565327167511, -2.007338285446167, -1.6672430038452148, -0.29643580317497253, 0.9007117748260498, -0.06763532012701035, -2.078707218170166, 0.11658883839845657, 1.7876547574996948, 0.1872405856847763, -0.5319313406944275, 0.378167062997818, -1.013427972793579, 3.1484878063201904, -1.5429593324661255, 0.672370970249176, -1.6586189270019531, 0.1415349394083023, -0.5258827805519104, 0.8967996835708618, -0.3060387372970581, 1.439342737197876, -1.6670314073562622, -1.00566828250885, 0.27699869871139526, 1.6274478435516357, 0.29713740944862366, 0.42946889996528625, -1.4270823001861572, 0.06918676942586899, -2.097381591796875, -1.010650396347046, -0.20759953558444977, 0.3976084887981415, 0.5717584490776062, -0.2961166203022003, 0.634975790977478, -1.0026532411575317, -0.8835422992706299, -1.600589394569397, 0.6874868869781494, 2.4308395385742188, -0.09795279800891876, 0.3059031069278717, 0.13476791977882385, -0.2318512350320816, -0.24081070721149445, -0.002210980048403144, -0.5007626414299011, -0.1948285847902298, 0.3569238781929016, 1.3185306787490845, -0.18599966168403625, 0.1894201934337616, 0.6132463216781616, -1.999625563621521, -0.39137783646583557, -1.1682031154632568, -0.7573682069778442, -0.06513028591871262]], [[1.1610807180404663, 0.005318419076502323, 0.5205300450325012, 0.7413027286529541, -1.797043800354004, 0.732356071472168, 0.48374104499816895, 0.39309120178222656, 0.2866845726966858, 0.9604052901268005, 0.15526893734931946, 1.3740177154541016, 0.22098194062709808, -1.3614742755889893, 0.007365422789007425, -0.8711652159690857, 0.46859514713287354, -0.6501113772392273, 0.011310956440865993, -2.076374053955078, 0.11300896108150482, 0.0966121256351471, 0.5412834882736206, -1.0005401372909546, 1.3985087871551514, 0.4019744396209717, 0.8648744225502014, -0.11547542363405228, -0.12926040589809418, -0.20138828456401825, 0.7550027370452881, -0.5417501330375671, 0.9982569217681885, -0.3862191140651703, -0.3780684769153595, 0.4547070264816284, 1.2317981719970703, -0.9838994145393372, 0.7332921028137207, 0.866224467754364, 1.310057520866394, -0.5629734396934509, 2.266310214996338, -0.25422680377960205, -0.18752393126487732, 0.799490213394165, 0.866707444190979, -0.2426072657108307, -0.13022522628307343, -0.09291639924049377, 0.43752360343933105, -0.764640748500824, -1.6411914825439453, -0.07410664856433868, 0.497384250164032, 0.462547242641449, -1.1113871335983276, 0.5406581163406372, 0.028073810040950775, 1.172421932220459, -0.5623722076416016, -0.9092110991477966, 0.9500689506530762, 1.3549482822418213]]], [[[-0.34037846326828003, -1.1643505096435547, -0.41661176085472107, 1.1973532438278198, 0.2593003809452057, -0.09078047424554825, -0.7356219291687012, -2.2562410831451416, 0.14389567077159882, -1.0534065961837769, -0.5635985732078552, 0.04960723593831062, 0.775377094745636, -0.48121580481529236, 0.9918762445449829, 1.9082280397415161, 0.7003567814826965, 1.125314712524414, -0.830521821975708, -0.03318123146891594, -0.4159076511859894, 0.005716696381568909, -0.0011184377362951636, -0.1715240627527237, -2.0795578956604004, -0.35095885396003723, 1.5835862159729004, -1.1027536392211914, -1.0942786931991577, 0.7534716129302979, -0.2276274710893631, 1.5794978141784668, 0.055470116436481476, 1.5237094163894653, 0.12997905910015106, 0.02015177346765995, -1.8545513153076172, 0.037858184427022934, -0.6928500533103943, 0.30853307247161865, -0.43223997950553894, -0.48217150568962097, -0.7173213958740234, 0.7787240147590637, 0.5337240099906921, -1.0684971809387207, -0.9664260745048523, -0.25863325595855713, -1.0337834358215332, 0.47283780574798584, -0.5161867141723633, 0.3763796389102936, 0.11513620615005493, 0.5427413582801819, -1.24397611618042, -0.8510127663612366, 1.212918758392334, -0.48554208874702454, 1.319474220275879, 0.8066354990005493, 0.7125495672225952, 2.126121759414673, 1.1164370775222778, -1.1550050973892212]], [[1.244177222251892, -1.5609349012374878, -0.014700428582727909, 0.023847470059990883, -0.7378594279289246, -1.0237842798233032, 0.7357694506645203, -0.3197748064994812, 0.2770327031612396, -1.5225008726119995, -0.40665575861930847, -0.6651167869567871, 0.5243309140205383, 1.2279689311981201, 0.16878584027290344, 1.5288140773773193, -1.4004722833633423, 0.04172775521874428, -1.0862528085708618, 0.7808581590652466, 1.0357142686843872, 0.7411890625953674, 1.7030127048492432, -0.1182507798075676, 0.9688611030578613, 1.673435091972351, -0.405636191368103, 1.0566306114196777, 0.36889180541038513, 0.4935324490070343, 0.5098717212677002, 1.0606403350830078, 1.0427896976470947, -1.5892866849899292, 1.805802345275879, -1.0805822610855103, -0.38528329133987427, 0.09025704860687256, 1.3948156833648682, 0.4826594293117523, 0.44564205408096313, 0.1202598288655281, -0.18483620882034302, -0.6667836308479309, 0.6521203517913818, 0.1918163299560547, 0.5847765803337097, -0.12214060127735138, -0.7577099204063416, 0.41970741748809814, -0.43191829323768616, 0.6332747340202332, 0.1298779547214508, 1.0364412069320679, -0.3560875356197357, 2.389754295349121, 0.9094513058662415, -1.6111561059951782, -0.28110387921333313, -2.1357333660125732, -2.2019500732421875, 0.46554338932037354, 0.49156731367111206, -0.4512139558792114]], [[-1.156217098236084, 1.5066142082214355, -0.08646980673074722, -0.5119755268096924, -1.4071986675262451, -0.08943633735179901, -0.0920383557677269, 0.011723233386874199, 1.4497960805892944, -0.7417081594467163, 2.3944895267486572, 1.971122145652771, 0.6181578636169434, -1.0305594205856323, -0.9335946440696716, -0.3519677221775055, 0.12784826755523682, -1.5964021682739258, -0.18257708847522736, -0.33858686685562134, 0.4184616506099701, 1.2082096338272095, -0.060660719871520996, 0.36649075150489807, -1.0595704317092896, -1.339651346206665, -0.06064685806632042, 0.1853167712688446, -1.0733637809753418, 0.10292116552591324, 1.022308588027954, -2.2884304523468018, -0.733651340007782, 1.6000657081604004, 0.0018690330907702446, 0.9229584336280823, -0.8816967606544495, -0.34499141573905945, -0.43603116273880005, 0.548014223575592, 0.6120191812515259, -0.20438003540039062, 0.5002683997154236, 0.3662923574447632, -0.17953874170780182, -0.8299857378005981, 0.9311074614524841, -2.1409120559692383, -1.1713253259658813, 0.012508250772953033, 0.5350683331489563, 2.8615152835845947, -0.055437151342630386, -0.12653864920139313, -0.8437842726707458, -0.04963362589478493, -0.1317971795797348, -1.1506539583206177, -0.7455146312713623, 0.3558819591999054, -0.7702377438545227, 0.32355642318725586, -0.11948097497224808, 0.017812497913837433]], [[0.2673974931240082, 1.3915255069732666, 0.7407464981079102, -0.4037449359893799, 0.3286682665348053, 1.7988651990890503, -0.05268010124564171, 0.9868413805961609, 0.05931561440229416, -1.1796917915344238, 1.5384982824325562, 0.6749634146690369, -0.36273112893104553, 1.1220369338989258, 1.544262170791626, -0.7048447728157043, 0.750203013420105, 0.8485108613967896, -0.39557746052742004, -0.13409222662448883, -0.4428395628929138, 1.5343388319015503, 0.7289606332778931, 0.4502353370189667, -0.45953696966171265, 0.8283787369728088, 0.2888459861278534, 0.758395254611969, -1.2632538080215454, -0.999126672744751, 0.6450923681259155, -0.16759490966796875, 0.369845986366272, -0.602825403213501, -0.6920320391654968, 0.8196582198143005, -0.5216243267059326, -1.5369179248809814, -0.43851473927497864, -0.9260963797569275, -0.6519713401794434, 0.3839642405509949, -0.6281265616416931, -0.5288437008857727, 1.451055645942688, 0.18507277965545654, -0.13895836472511292, 0.6585877537727356, 1.4199190139770508, 1.3154404163360596, 0.8737612366676331, 1.3185343742370605, 0.6920467019081116, 1.029231071472168, 1.6560025215148926, -1.2752419710159302, 0.014149273745715618, -0.9259061813354492, 0.3197450339794159, 0.9105768203735352, 1.0447567701339722, 0.8399017453193665, 1.0138843059539795, 0.8497070074081421]]]], "dtype": "float32"}, {"__nd__": [0, 7, 15], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[-0.7392309308052063, 1.0613659620285034, -1.8418433666229248, -0.5448198318481445, -0.8449132442474365, 1.5745280981063843, 0.7666898965835571, 1.1683930158615112, -0.9695287346839905, 0.6420978307723999, -1.1271424293518066, 1.9708565473556519, -0.8387977480888367, -0.8470180630683899, -0.1867648810148239, -0.7236369252204895, 0.4150848388671875, -2.594101905822754, -1.4291977882385254, -0.49087169766426086, 0.607330322265625, -1.8257704973220825, 1.9168661832809448, -0.9942984580993652, 0.9511772394180298, 1.1189039945602417, 1.0379713773727417, -1.8082610368728638, -0.09366113692522049, 0.7770522236824036, -1.3662649393081665, -0.6099796891212463, 0.3429250121116638, 0.035387855023145676, -0.301724374294281, 0.2977226674556732, -1.6424857378005981, 0.49570050835609436, 0.2616593539714813, -1.4863725900650024, 1.4190727472305298, 0.4760321378707886, -1.540210485458374, 0.45527854561805725, -0.4815206229686737, 1.239449381828308, 0.37637120485305786, 0.9681996703147888, 1.2040401697158813, -1.233989953994751, 0.0832611545920372, 1.0115677118301392, 1.5188175439834595, 1.0827977657318115, 0.20136523246765137, 0.2714795768260956, 1.0787492990493774, -0.3876892030239105, 1.4943758249282837, -0.007025897968560457, -1.1131606101989746, -0.6851240396499634, 1.3054429292678833, -0.0714552104473114], [0.29444772005081177, 0.9621285796165466, 0.9284215569496155, 1.3320931196212769, 0.806363046169281, -0.33560803532600403, -0.05426894500851631, 0.49421510100364685, -0.3102922737598419, 0.5494387745857239, 1.1536095142364502, -0.8012459874153137, -1.7703840732574463, 1.193655014038086, 0.017759909853339195, 0.7086698412895203, -1.3323053121566772, 1.7329925298690796, 0.738619863986969, -0.1260741800069809, -0.3519880473613739, 0.11389587819576263, -0.1667313426733017, -0.2786901295185089, -1.1195999383926392, 1.2628093957901, -0.03957768529653549, 0.07463563978672028, 0.09291598945856094, -2.358217239379883, 0.5271414518356323, -0.32915493845939636, -0.2892284691333771, 0.0697878748178482, -0.9724755883216858, 0.8433808088302612, -0.6958646178245544, -0.9036580324172974, -0.8621895909309387, 0.1587526947259903, 0.7457794547080994, -0.5950532555580139, -1.5101866722106934, 1.062490701675415, -1.065083622932434, 1.1948304176330566, -0.05429587513208389, -0.6983498930931091, -0.6400399208068848, 0.461866170167923, 0.8893718123435974, -0.1981702297925949, 0.9469344019889832, 2.0869781970977783, -0.20830833911895752, -1.1494466066360474, -1.0544003248214722, -1.3903353214263916, -0.639665424823761, -0.7237792611122131, -1.3251032829284668, 0.530184268951416, -2.6113390922546387, -1.3207989931106567], [2.015251874923706, 1.0378024578094482, 0.5227916836738586, 0.45725905895233154, 1.770937204360962, 0.5356296300888062, 0.07757708430290222, -1.230160117149353, -0.07366681843996048, -0.7338839769363403, -0.7061752676963806, -0.7142389416694641, -0.044646862894296646, -0.32971081137657166, 0.8711104989051819, -1.326181411743164, -0.1694701910018921, 1.8594975471496582, 0.03889733552932739, 1.1986151933670044, 1.1368634700775146, 1.3166477680206299, 0.7381191849708557, -1.3804460763931274, -0.267387717962265, -0.38033556938171387, -1.5488553047180176, -0.46388161182403564, -1.250079870223999, -1.6655514240264893, 0.21089625358581543, -0.043591272085905075, -0.6878148913383484, -0.9505863785743713, -0.5412734150886536, -0.3940903842449188, -0.40811029076576233, -0.052707746624946594, -0.43554067611694336, 0.38697198033332825, 0.5228708982467651, 0.3578696846961975, -0.6631035208702087, -0.6165061593055725, 0.4348641037940979, -0.197042778134346, -0.04315171018242836, -2.491995334625244, -2.700103998184204, 0.33714818954467773, 0.32955867052078247, 1.3934319019317627, 0.7211630940437317, 0.221738800406456, 0.01636245660483837, -1.4267404079437256, 0.8831759095191956, -0.6035860180854797, 0.5024844408035278, 0.20004165172576904, 0.27209702134132385, 0.8926752209663391, -0.48986610770225525, -0.38723519444465637], [-0.855861485004425, -0.3771520256996155, 1.665940761566162, -0.36331865191459656, 1.1856162548065186, 0.8260114192962646, 0.7292817831039429, 0.8030151128768921, -1.0142765045166016, 0.3734476864337921, 1.5884120464324951, -0.6957959532737732, -0.04710708558559418, -0.6891591548919678, 0.3170115053653717, -1.0785300731658936, 1.6096798181533813, -0.37683477997779846, -1.5426615476608276, 1.7907909154891968, 0.9705228805541992, 0.5185114741325378, 0.5325875282287598, -0.3577980101108551, -0.22329559922218323, 0.7240192294120789, 0.9176249504089355, 0.932612419128418, 0.21255098283290863, -0.46638810634613037, -0.192087322473526, 1.1095997095108032, 1.4399181604385376, -0.34131237864494324, -0.47730401158332825, 1.2475496530532837, 0.7236490249633789, 0.6972581744194031, -1.3123337030410767, -0.3737574517726898, -0.5149955153465271, 0.207904651761055, -1.6849420070648193, 1.2717528343200684, 0.30013179779052734, 0.4348370134830475, -1.1841403245925903, -0.02642037160694599, 1.3346502780914307, -1.5696955919265747, 0.30576804280281067, 0.6988049149513245, -0.6919471025466919, 0.8147412538528442, -0.33006903529167175, 1.229766607284546, 0.5275117754936218, 0.22436589002609253, 2.008915901184082, 0.8318964838981628, 1.3086357116699219, -1.1895196437835693, -0.6466070413589478, 0.8365690112113953], [0.9056118726730347, -0.055740028619766235, 0.779726505279541, -1.0583537817001343, 0.2138352245092392, -1.147283911705017, 0.3405705392360687, -0.4861719012260437, -0.6409975290298462, 0.056299030780792236, 1.817460060119629, -1.0117617845535278, -0.6017506122589111, 1.0676641464233398, -0.42356666922569275, 1.7692642211914062, 0.22252970933914185, 0.09063191711902618, 0.3568841516971588, 1.7411795854568481, 0.655361533164978, 0.5167121291160583, 0.2518577575683594, -1.562418818473816, -0.8852720260620117, 1.4833440780639648, 0.8218783140182495, 0.5060648322105408, 0.6370627284049988, -0.5435930490493774, 0.33796426653862, -0.9007946252822876, 0.3913623094558716, -0.7141351103782654, -0.21381983160972595, 0.4534148573875427, -0.32678771018981934, 0.647581934928894, 0.4560382664203644, 0.35945814847946167, 1.324411392211914, -1.4738847017288208, -1.5962822437286377, 0.6162418127059937, 0.9407941102981567, 2.0116403102874756, 0.7562568187713623, 0.8302149772644043, -0.8830800652503967, 0.7526217699050903, 0.6738389134407043, -1.0558651685714722, -0.8963174819946289, -0.11103120446205139, 0.2882537543773651, -2.9447855949401855, 0.07030551135540009, 0.23085449635982513, -0.17683635652065277, 0.2668249011039734, 0.4559062421321869, -1.0116380453109741, 0.41618016362190247, -0.2513071894645691], [-0.945878803730011, 0.20914079248905182, 1.4876441955566406, -1.4601155519485474, -1.6531099081039429, -1.6531661748886108, 1.9773117303848267, -0.2515837550163269, -0.0594867542386055, -0.7865757346153259, -0.9404478073120117, -2.026860237121582, 0.1704062968492508, 0.2595598101615906, -1.9765340089797974, -1.0431492328643799, -0.7441137433052063, -0.02600245364010334, 0.24412916600704193, 1.1257929801940918, 1.364456057548523, 1.0684545040130615, 1.3415285348892212, -1.1664166450500488, -0.5544917583465576, 1.1436097621917725, 0.28866109251976013, 0.018721718341112137, -1.9789881706237793, -1.8592429161071777, 0.7308894395828247, 0.34157639741897583, -0.2960576117038727, -0.41297173500061035, -1.726102590560913, 0.3712151348590851, 1.0344208478927612, 1.642171025276184, 0.04913794994354248, -0.20419667661190033, -1.0959970951080322, 1.2232437133789062, 1.7068555355072021, -0.6131700277328491, 0.29075875878334045, 0.09067010879516602, -0.57382732629776, -2.4974496364593506, -0.36599910259246826, 0.607444167137146, -1.443302035331726, -0.838881254196167, -3.2835605144500732, 1.2845276594161987, 0.5002763867378235, 0.04547853022813797, -0.2704733908176422, -0.637283205986023, -0.07880625128746033, -0.31031492352485657, -0.24835671484470367, -0.033811114728450775, 0.12565097212791443, 1.5489784479141235], [-1.449432373046875, 0.26196861267089844, -0.6245608329772949, -1.1394811868667603, -2.368170738220215, 0.6068921685218811, -0.9903792142868042, -0.7596859931945801, 0.45987868309020996, -1.093472957611084, 0.016045130789279938, -0.08488040417432785, 0.7918434143066406, 1.0006409883499146, -0.8768518567085266, 0.907177746295929, -0.1067456603050232, 0.1127975657582283, -0.2225136160850525, 0.7655396461486816, -1.2222003936767578, 0.2500842809677124, 1.1657798290252686, -0.2936699092388153, 2.0177724361419678, -0.745905876159668, 0.2036961317062378, -0.49260851740837097, -0.6287503242492676, -1.1257710456848145, -0.5500664710998535, -0.06223707273602486, -0.8130971789360046, 0.8723957538604736, 1.510326623916626, -1.5388962030410767, 0.8248433470726013, 0.17806008458137512, 0.2333207130432129, 2.4504992961883545, 1.1943246126174927, 0.3243098556995392, 0.17311415076255798, -2.165717124938965, 0.7723909020423889, -0.7486869692802429, 0.17573262751102448, 0.09763062745332718, 0.9603515267372131, 2.0305027961730957, 0.9836022853851318, 1.2729486227035522, -0.23454320430755615, 0.4557735025882721, 0.769248902797699, -1.43962824344635, 2.0415430068969727, -0.7148682475090027, -1.277543306350708, 1.6340441703796387, 2.118438720703125, -0.7031057476997375, -0.6133099794387817, 3.111912965774536], [0.1126171126961708, -1.984058141708374, -0.6122687458992004, 2.0021188259124756, 0.6591240763664246, 0.7520990967750549, 1.2245573997497559, 0.6757786870002747, -0.3332044780254364, -1.5943117141723633, 0.44637084007263184, -1.1160348653793335, -0.05623851716518402, 1.758971095085144, 0.5406503081321716, 0.05949153006076813, 0.06568005681037903, 0.730792760848999, 0.9212337732315063, -1.3168138265609741, 0.1854793280363083, 1.7156471014022827, 0.7543686628341675, -0.7651258707046509, -0.6796277165412903, -2.061922550201416, -0.7933788299560547, 1.6069653034210205, -0.0695239007472992, -0.855010986328125, -0.5761204957962036, -0.11074551194906235, -0.10654617100954056, -0.35397371649742126, -0.7690944671630859, 0.6365700960159302, 1.5385345220565796, -0.5679569840431213, -0.5412718653678894, 1.189353108406067, -1.4637436866760254, -0.5913947224617004, -0.3780970871448517, 0.16778159141540527, -0.27493882179260254, -0.8866033554077148, 0.31050634384155273, -1.1877104043960571, -1.742513656616211, -2.0282576084136963, 0.7486985325813293, -1.7368555068969727, 0.4470572769641876, 0.6707091331481934, 0.8068993091583252, -0.17468272149562836, -0.5482001304626465, 0.0856713354587555, -0.5937349200248718, -1.286202311515808, 1.2554398775100708, -0.8944374322891235, 1.7958890199661255, 0.5147645473480225], [-0.45455676317214966, 0.43764764070510864, 1.230149507522583, -1.3784692287445068, -0.926772952079773, -1.0492675304412842, 0.16503140330314636, 0.4042508602142334, 0.028968343511223793, -0.7282098531723022, -0.5284513235092163, 1.187043309211731, 1.1169426441192627, 0.13135528564453125, -1.310608983039856, 0.6281268000602722, 0.2876591682434082, 0.25273048877716064, 1.7300316095352173, 0.8205397129058838, -0.974485456943512, -0.9647605419158936, 1.3803138732910156, 0.20571652054786682, 0.8870580196380615, -0.10639959573745728, -0.9494908452033997, 0.32420140504837036, 1.6189154386520386, 0.39913585782051086, -1.065564751625061, -0.24585874378681183, -0.42841270565986633, 0.2473440170288086, 1.0980541706085205, 2.0542898178100586, 0.3144912123680115, -0.3364866077899933, 0.1715981662273407, 0.11086616665124893, -0.32979634404182434, 0.1186041608452797, 0.09529157727956772, -1.0805187225341797, 0.6438245177268982, -0.2667025625705719, -0.8877440690994263, 0.5859832167625427, -0.39159297943115234, 1.8382551670074463, 0.5709066987037659, 0.1686367392539978, -0.665776252746582, -0.31839120388031006, 0.9355282783508301, -0.6024860739707947, 0.8436412811279297, -2.0179641246795654, 0.28045257925987244, 0.8953021764755249, 0.41629400849342346, 0.07274018973112106, 1.086340069770813, -0.7792912125587463], [-0.9079190492630005, 0.1802140772342682, 1.2776236534118652, 0.5065013766288757, -0.11938237398862839, 0.37507161498069763, 0.5171626806259155, -1.1704579591751099, -2.7371013164520264, -0.950610339641571, -1.4196820259094238, -0.5050193071365356, -1.1275153160095215, -0.7912658452987671, 0.5087776780128479, 0.17351014912128448, 0.05337090045213699, 1.7401034832000732, -0.3785295784473419, -0.2968817353248596, 1.3511508703231812, -1.025349497795105, -1.5425773859024048, 1.397329568862915, 0.42847737669944763, 0.6939471364021301, 0.7324030995368958, 1.000512719154358, -0.37342125177383423, 0.024766210466623306, -1.0105915069580078, 0.24289265275001526, 0.3374541997909546, -0.3463873863220215, 0.7765704989433289, 3.064624071121216, -1.0395714044570923, 0.9449557662010193, 1.060904860496521, 1.186570167541504, -0.6814988851547241, 1.3214032649993896, 0.7814787030220032, 0.004755706060677767, -0.21026098728179932, 0.3801894783973694, 1.071524977684021, 1.024174451828003, 2.4328603744506836, -0.2064686417579651, 0.7699827551841736, -0.2774447500705719, 0.9362072348594666, 0.28631535172462463, -0.44236698746681213, -1.170732021331787, -0.6131279468536377, 0.3971872627735138, 1.6269680261611938, 0.9308051466941833, -1.0413693189620972, -0.33361268043518066, 0.1840648353099823, 0.01132616400718689], [0.19017614424228668, 0.8913050889968872, 0.31386786699295044, -0.601158618927002, -1.8446869850158691, -0.2656921446323395, -1.6127432584762573, -0.10446330159902573, 0.363433837890625, -1.2998757362365723, -0.11584888398647308, -0.3271665871143341, 1.906078815460205, 0.44876807928085327, -1.9007844924926758, 0.42046672105789185, 0.3862980902194977, -0.2349327951669693, -0.4137638509273529, 0.5544675588607788, 0.6138501167297363, 0.38203006982803345, -1.0521268844604492, -0.3142268657684326, -1.6254591941833496, -1.566889762878418, -0.6347933411598206, -1.1325814723968506, -0.4956102967262268, 2.3927721977233887, 0.4025426506996155, -0.9809014201164246, -2.2614598274230957, -0.9428343772888184, -1.5433629751205444, -2.3115298748016357, 0.6050446629524231, -0.6025458574295044, 0.3164512813091278, 2.966238260269165, 0.14657582342624664, -0.0074188197031617165, 0.22416597604751587, -0.391497403383255, -0.9520021677017212, -0.3374938666820526, 1.0182347297668457, -0.3933661878108978, -1.492048978805542, -1.368557095527649, -1.1497435569763184, -0.1072094589471817, 0.3657081127166748, 0.36833882331848145, 1.0810695886611938, -0.30665305256843567, 0.7394214272499084, 0.23684793710708618, 0.6244451403617859, -1.9157264232635498, 1.1791285276412964, -1.6059455871582031, -1.1812087297439575, 0.728539764881134], [-0.43165335059165955, 1.728952169418335, 0.12705281376838684, -1.0572758913040161, 0.128393292427063, -0.42599353194236755, 1.5300532579421997, 1.6091804504394531, 0.7201277017593384, -0.42249882221221924, 0.06931523978710175, 1.0352593660354614, -0.8136550784111023, -0.8067298531532288, 0.8386394381523132, 0.7259921431541443, -0.2420131117105484, -0.2782663106918335, -1.9611250162124634, 1.173648715019226, -0.1261032074689865, -1.4362707138061523, -1.1350212097167969, -0.9003010988235474, 1.6969619989395142, 0.6850574612617493, -1.341442346572876, 1.8512842655181885, 1.3351792097091675, -1.0612409114837646, -2.66099214553833, 0.9614986181259155, -0.6027781367301941, -0.13628479838371277, -0.524908185005188, 1.2592880725860596, 1.4789767265319824, 0.810117781162262, -0.15063685178756714, 0.26184359192848206, 0.6211815476417542, -1.5746363401412964, 1.2433815002441406, -0.31869974732398987, 0.26307061314582825, 0.14000190794467926, 0.9083163738250732, 1.0714818239212036, 2.2131998538970947, -0.34747856855392456, -0.8965520858764648, 0.14481224119663239, -0.6418266892433167, 0.8751027584075928, -0.7909221649169922, 0.15036310255527496, 0.9588245153427124, 1.034327507019043, 0.22643126547336578, 1.4341362714767456, 1.3108830451965332, -1.6749932765960693, 0.3853166103363037, 0.20764310657978058], [-0.2698063850402832, -0.607272744178772, 0.07507789134979248, 0.41873955726623535, 0.3086544871330261, 1.3998866081237793, 1.0281139612197876, -0.5532593727111816, -0.7224798202514648, 0.8842297196388245, -0.6599516868591309, 0.20378567278385162, 0.5136725306510925, 1.0678472518920898, 0.5548705458641052, 1.483630657196045, -0.41991153359413147, 0.7100406289100647, -0.6179177165031433, 0.22685877978801727, 0.16443012654781342, -0.8879100680351257, -2.5632784366607666, 0.9970925450325012, 0.4042292833328247, 0.18224069476127625, -0.6591445803642273, -0.29145991802215576, 0.4969172775745392, 0.06155714392662048, 1.200844168663025, 0.1582016795873642, -0.7803479433059692, 0.6506232023239136, -0.593580424785614, 0.2554992139339447, -1.7722288370132446, -0.31255680322647095, 0.5930131077766418, -0.6167822480201721, 0.005562063306570053, 0.6214967966079712, 0.09812398999929428, 1.8157238960266113, -0.22971957921981812, -0.15062332153320312, -0.13217906653881073, -1.9484788179397583, -1.505789041519165, 0.5606258511543274, -0.30178529024124146, 1.41615629196167, -0.05088788643479347, -0.1429254710674286, 1.0557904243469238, 0.3035111725330353, 1.6309014558792114, -1.0298982858657837, 0.46728771924972534, 0.7073956727981567, 0.5844882130622864, 1.1814675331115723, 1.0607037544250488, 0.0700731948018074], [-0.5216994285583496, 1.0194790363311768, 0.0929674580693245, 1.0061924457550049, 1.4043821096420288, 0.25849786400794983, -0.8391388058662415, -0.188700869679451, 0.8577681183815002, -0.16371336579322815, 1.2553577423095703, 0.12203249335289001, 0.21839630603790283, -0.5367452502250671, -0.008135730400681496, 1.1862748861312866, -0.25078389048576355, -0.24314925074577332, -0.6909047365188599, 0.8662309646606445, -1.438070297241211, -0.5731570720672607, 0.9874779582023621, 1.4973156452178955, -0.1496429741382599, 1.033983588218689, -1.914989709854126, 0.8238314390182495, 0.35727834701538086, -1.1592878103256226, -1.716760277748108, -0.44351816177368164, 0.32016366720199585, 0.4285535216331482, 0.17032885551452637, 0.8300293684005737, 0.7591947317123413, -1.325154423713684, -0.3256736993789673, -0.4546818733215332, 1.082566738128662, -0.4368080496788025, 1.0607962608337402, 0.14688044786453247, 2.4004673957824707, 0.005745717324316502, 0.059517305344343185, -0.056776802986860275, 0.20182116329669952, 2.482877731323242, -0.4495837688446045, -0.8182248473167419, -0.14498426020145416, 0.7236251831054688, -0.9369282126426697, 0.8460574746131897, 0.8127056360244751, -0.1168634369969368, -0.2049449384212494, -0.7416505217552185, -0.22885537147521973, -0.7548636794090271, 1.2864465713500977, 1.6693135499954224], [-0.24161919951438904, 0.00040231773164123297, -1.8577793836593628, -0.48929232358932495, 0.3772006034851074, 0.3102934658527374, 0.15976007282733917, 0.880998969078064, 0.1265142410993576, -0.35277992486953735, 1.6342995166778564, -1.8823978900909424, -1.1231029033660889, -0.7060480117797852, -0.6988998055458069, -0.7454692721366882, -0.19863399863243103, -0.6876280903816223, -0.03457644209265709, 0.6096417307853699, 0.046772900968790054, 0.14978359639644623, -0.9177576303482056, -1.1184498071670532, -0.2779383361339569, 0.936154842376709, 0.3869842290878296, 1.095211386680603, 1.0011459589004517, 1.130170226097107, -0.793280839920044, 1.2445451021194458, 0.765385091304779, 2.0772807598114014, 0.597710132598877, 0.3354604244232178, -0.1174328625202179, 0.1754874438047409, -1.6325438022613525, -0.23683799803256989, -0.026065850630402565, 0.9965258836746216, -0.6405432224273682, 0.14371009171009064, 1.8275507688522339, -0.5140311121940613, 0.8498762845993042, -0.5180673003196716, -0.08717534691095352, 0.46998757123947144, 0.8892123699188232, -0.4767798185348511, -1.0020980834960938, 1.5188612937927246, 0.19799013435840607, 0.8002609014511108, 0.9185786843299866, 0.7643093466758728, -0.24615730345249176, -0.12190839648246765, -1.5274707078933716, 0.1913345605134964, -0.4519769847393036, -0.009605282917618752], [1.86375093460083, 0.6937499642372131, 0.06547141075134277, -1.3327445983886719, 0.7337110638618469, 1.5181533098220825, -0.8004674315452576, 0.43257465958595276, -0.49807146191596985, -0.1366279423236847, -0.3479275703430176, -1.5764728784561157, -0.49307113885879517, -1.5852549076080322, 1.3782265186309814, 1.0388894081115723, 0.05506785586476326, 0.2867772877216339, 0.0687292069196701, 0.20761720836162567, 0.20490533113479614, -0.22359073162078857, -0.15187609195709229, -1.3104325532913208, -0.9730347990989685, -0.5838696360588074, 0.21642760932445526, -0.3533463776111603, 1.885884165763855, -0.173744335770607, -0.337854266166687, 0.4379376769065857, 1.0245460271835327, 1.833386778831482, 0.9708805084228516, -0.5633888840675354, -0.15201693773269653, 1.0467700958251953, 0.26900050044059753, 0.8014563322067261, -0.05585052818059921, 1.5882296562194824, 1.437668800354004, 1.1231521368026733, 0.1743306666612625, -1.5570294857025146, -1.299076795578003, 0.3423963189125061, 0.6638202667236328, -2.8262147903442383, -0.4391079843044281, 1.1521999835968018, 0.9599878787994385, -0.4598434567451477, 0.6500676274299622, 0.057154640555381775, -0.185583233833313, 1.1234819889068604, 1.1430305242538452, -0.4834875762462616, -0.8938437700271606, -1.2477530241012573, -1.35883367061615, 1.3349007368087769]], [[0.3983820378780365, -1.3772249221801758, 2.465254068374634, -0.28950127959251404, -0.1404246687889099, -0.3630053400993347, -0.1709207445383072, -0.041876327246427536, 0.3634560704231262, -0.4820344150066376, -1.228871464729309, -0.44429469108581543, -1.2454125881195068, 0.6777396202087402, -1.2521214485168457, -0.2776854336261749, -0.45383980870246887, -1.130686640739441, 0.8865922689437866, 0.2696767747402191, -0.5439797639846802, 0.8442064523696899, 0.2793535888195038, -0.1931031048297882, 0.1057659238576889, -0.8760663866996765, 0.7355189323425293, -0.12479784339666367, -1.1539658308029175, 1.5380021333694458, -0.49970757961273193, -1.4923601150512695, -0.6619400978088379, -0.16763611137866974, 1.7743306159973145, -0.2124159336090088, -0.1899750828742981, -1.1723887920379639, -0.6436547040939331, -0.45482534170150757, -0.5261010527610779, 0.6517183780670166, 0.3937457203865051, 0.9239131212234497, 0.05636700242757797, 2.458667278289795, 0.5549972653388977, 1.2331023216247559, 0.0352318175137043, -1.6085805892944336, -0.6082191467285156, 0.7990738749504089, -0.6824512481689453, 1.1453282833099365, 0.958700954914093, -0.29311999678611755, -0.10845762491226196, 1.1025612354278564, -0.6072331070899963, -0.20503880083560944, 1.6333868503570557, -0.3453291058540344, -0.12001066654920578, -0.5380480885505676], [-0.4019356966018677, 0.1071111410856247, 0.3789922595024109, -1.3443832397460938, -1.4471772909164429, 0.20568685233592987, -0.03210219740867615, -1.9469980001449585, -1.6273365020751953, -0.7208101153373718, 0.33340221643447876, 0.22141604125499725, 0.999830424785614, 1.159284234046936, 2.3200325965881348, -1.3723598718643188, -0.013785125687718391, 1.0572185516357422, -1.1914669275283813, -1.340640902519226, 0.21681663393974304, 1.0876883268356323, 1.2406551837921143, -1.6978176832199097, -1.151814579963684, -0.06735824048519135, -0.48677846789360046, 0.3394567668437958, 0.15675535798072815, 1.7359110116958618, -0.033442456275224686, 0.2049727439880371, 0.10196290910243988, 1.1016439199447632, -1.0495768785476685, 0.4038074314594269, -0.7196693420410156, -1.3453353643417358, 1.1762868165969849, 0.5426834225654602, 1.3904682397842407, 0.2827655076980591, 0.6671103239059448, 0.16937315464019775, 0.5917483568191528, -0.3609093725681305, 1.701602578163147, -0.8201969861984253, 1.7057604789733887, -0.1485859602689743, 0.8921769261360168, 0.18080472946166992, 0.4822855293750763, 1.4990304708480835, 0.18873842060565948, -0.8698955178260803, 0.2848009467124939, 0.684927225112915, -2.343674421310425, 1.005519986152649, 1.1186068058013916, -1.675768494606018, -0.09786126762628555, 0.8562725186347961], [-1.0792711973190308, -0.3587975800037384, -0.27950209379196167, 0.6004593372344971, 1.143613576889038, 0.02294977754354477, 0.1153683140873909, 0.9619436860084534, 0.5853611826896667, -0.05394911393523216, 1.3585437536239624, -1.3352926969528198, 0.2020653635263443, -0.3604046702384949, -0.44249820709228516, 0.24635453522205353, 0.08954466134309769, 2.466902732849121, 0.6843624114990234, 0.3969602584838867, 1.674640417098999, 1.0493732690811157, -1.7812647819519043, 0.17044976353645325, 0.781863272190094, -0.5458417534828186, 0.33988142013549805, -0.4015931189060211, 0.4161466658115387, -1.5575698614120483, -1.874250888824463, -0.4973315894603729, 0.26492151618003845, 1.3531397581100464, -0.232381671667099, -0.6981062293052673, 0.9420226812362671, 0.07280247658491135, 1.8154182434082031, -0.0507391132414341, -1.8845176696777344, -0.7511929869651794, -0.9051104784011841, -1.3399128913879395, -0.7324886918067932, -0.6487211585044861, -0.25051265954971313, -0.5116637349128723, 0.4315355718135834, 0.4358985722064972, 0.20217981934547424, 0.22221021354198456, -1.8342421054840088, -1.6215094327926636, -0.8437504768371582, 0.27673444151878357, -1.4215724468231201, -0.3913985788822174, 0.2697241008281708, 1.9867075681686401, 0.28176867961883545, -0.5059260725975037, 1.1024010181427002, 0.22433927655220032], [-0.27569183707237244, 1.1198322772979736, -0.8906885981559753, 0.2054339051246643, 0.09374996274709702, 0.9072456955909729, -1.1242132186889648, -1.2728910446166992, -0.37835124135017395, -0.2663916349411011, 0.04562491178512573, -0.3687709867954254, 0.42910754680633545, 0.6845958828926086, 0.22120583057403564, -1.0464434623718262, 1.1411778926849365, -1.5240790843963623, -0.20895320177078247, 0.8774041533470154, -0.6181960701942444, 1.2746925354003906, 1.8907825946807861, 1.5551873445510864, 0.42311325669288635, 0.9650139212608337, -0.5183897614479065, 0.3883185386657715, 0.4431073069572449, 0.0834994912147522, 0.14439734816551208, -0.21514803171157837, -1.957853078842163, 0.38814401626586914, 0.3022112250328064, 2.0293054580688477, 0.6977156400680542, 0.058108072727918625, 0.8594762086868286, 0.6098951697349548, -0.025558190420269966, -0.3873504102230072, -1.4679991006851196, 0.11340352892875671, 1.6074281930923462, -1.8636541366577148, -0.9536433219909668, 0.7432786226272583, 0.18359945714473724, 0.4093780219554901, -1.857773780822754, -0.5945903658866882, 0.4945397675037384, 0.5778988003730774, 0.28581249713897705, -0.8860844969749451, 0.5290326476097107, -1.1968166828155518, -0.9440393447875977, -1.1677287817001343, 1.425971269607544, -0.41940295696258545, 0.046004146337509155, -2.1984241008758545], [1.0813047885894775, 0.45218610763549805, -1.6580231189727783, -1.609822154045105, 1.6429228782653809, -0.9511963725090027, 1.8240394592285156, -0.4779016375541687, -2.4858362674713135, 1.09334135055542, 0.4894861876964569, 0.6681703925132751, -0.5189090371131897, -0.22533050179481506, 0.9421473741531372, -0.8329755663871765, -1.975229024887085, 0.8923466801643372, -0.6322474479675293, 1.2579083442687988, -0.49562975764274597, -0.44075384736061096, -0.26612457633018494, -2.3663153648376465, 1.1586400270462036, -0.6432483196258545, -0.21072284877300262, 1.4466402530670166, 0.09996433556079865, 0.8790737986564636, 0.3202430307865143, 0.7567090392112732, 0.2622949481010437, 1.1387934684753418, -0.22413885593414307, 0.4465140104293823, -0.7214215993881226, 1.1713738441467285, -2.1937522888183594, 0.17275454103946686, 0.3479059636592865, 0.32904183864593506, -0.01979093626141548, 1.3260551691055298, 2.101428985595703, -0.6073958277702332, -1.9200876951217651, -1.0991789102554321, -0.8243827819824219, -0.148102268576622, -0.06763698905706406, 1.0654654502868652, -0.43930330872535706, -0.04962938278913498, 0.09713341295719147, -0.8906799554824829, -0.3303353190422058, 2.742591381072998, -0.10513272136449814, -0.975318193435669, -2.1901962757110596, 1.0241239070892334, -0.335629940032959, -0.6474533081054688], [2.4308433532714844, 1.4182775020599365, -1.7253302335739136, -0.3614228069782257, 0.9838337302207947, 0.8207517862319946, -1.127099871635437, 1.762094259262085, 1.835807204246521, -1.122765302658081, -1.7803300619125366, -1.405688762664795, 0.6507818102836609, 1.0621857643127441, 0.16741704940795898, -0.6129648089408875, -0.1111510768532753, 0.5452548861503601, -1.2326768636703491, -1.7168357372283936, -0.105614073574543, 1.2133082151412964, 0.16984093189239502, 0.17263193428516388, -0.6962419748306274, 0.8977471590042114, -1.7302011251449585, -0.0884302407503128, -0.6173286437988281, -0.17867448925971985, -0.9521907567977905, 0.7226251363754272, -0.10454924404621124, -0.6244663000106812, 0.48045769333839417, -0.08159482479095459, -0.5501055717468262, -0.6215181350708008, 0.7462332844734192, -0.9719409346580505, -0.25569018721580505, -0.3024829626083374, 0.2788255214691162, -0.2284586876630783, 2.1212375164031982, -1.7823542356491089, 0.5563126802444458, -0.06899502873420715, 1.0719588994979858, -1.661516785621643, 1.1792610883712769, -1.4204144477844238, 0.6526589393615723, 0.4729805290699005, -0.007830378599464893, -0.7760050892829895, -0.8554891347885132, 0.05861421674489975, -0.7905116081237793, -1.9903665781021118, -1.847069501876831, 0.7606856226921082, 0.785500705242157, 0.47872257232666016], [0.012259075418114662, 0.3462595045566559, 0.46212875843048096, 0.21987619996070862, -0.8225317597389221, -0.7628558874130249, -1.1806305646896362, 0.12368922680616379, -1.0183985233306885, -0.18521061539649963, 1.2895960807800293, -0.4007664918899536, 0.9477011561393738, 1.5366467237472534, -1.8462074995040894, -1.4550038576126099, 0.4535783529281616, -0.727559506893158, 0.5174764394760132, 0.6908771395683289, -0.5290382504463196, 0.8123423457145691, 1.7308964729309082, 0.9444819688796997, -0.8852678537368774, -0.15462060272693634, -0.5446020364761353, 1.5613423585891724, -0.19860419631004333, -0.6944221258163452, 0.04367179423570633, 0.5124685764312744, -0.37633129954338074, 0.10705791413784027, 0.0024917887058109045, 0.7415816783905029, 0.9778999090194702, 0.27968519926071167, -0.4603579342365265, 0.4139566421508789, -0.8825797438621521, 1.0847476720809937, 0.36036601662635803, 0.2574433386325836, -2.4102981090545654, 2.384186029434204, 1.4711592197418213, 1.288474440574646, 0.147059366106987, 0.6938448548316956, 2.8812804222106934, -0.6559854745864868, 0.7759380340576172, 0.6342946887016296, 0.17647245526313782, 0.3914646506309509, -0.9390909075737, 0.6915391087532043, -1.263176679611206, 1.8821666240692139, 0.8521352410316467, -0.7967486381530762, -2.0363402366638184, 0.16343924403190613], [0.5680439472198486, -0.9327842593193054, 1.741498589515686, 0.3390537202358246, -0.39467480778694153, -1.3252567052841187, -0.04735438898205757, -0.12439973652362823, 0.7260642051696777, 1.125786304473877, 0.10802280902862549, 0.5389137864112854, 1.5030630826950073, 0.6967734694480896, -0.17905555665493011, 0.6983010768890381, -0.44238340854644775, 0.5312957167625427, 0.13035549223423004, -1.0187915563583374, 0.6830295920372009, 0.16322802007198334, 0.8106619119644165, -1.24861478805542, 1.18617582321167, 1.2288119792938232, -0.2741584777832031, -0.022449886426329613, 0.12253140658140182, 0.052425842732191086, 0.9073825478553772, -0.6430671811103821, -1.0808987617492676, -0.1760164350271225, 1.962971568107605, -0.3276088237762451, -0.9405286908149719, -0.8623668551445007, 0.22092878818511963, 1.228193759918213, -0.6862431168556213, 0.22234377264976501, 0.2604827582836151, 0.5109481811523438, 0.18895459175109863, 1.0941689014434814, 0.7297549843788147, -0.5733395218849182, 2.4672462940216064, -1.3218557834625244, -1.2772600650787354, 1.0755003690719604, 0.22601525485515594, -0.20893940329551697, 0.23949386179447174, 0.724484384059906, -0.38553953170776367, -0.5983303189277649, -1.0125545263290405, -0.8265764117240906, -0.7659786343574524, 1.1788495779037476, 0.3116062581539154, 0.5756569504737854], [-0.5320655107498169, 1.0790150165557861, -1.5308595895767212, 0.5036996603012085, 0.7234464287757874, -0.5512312650680542, -0.9989850521087646, 0.5466575026512146, -0.684355616569519, -0.8702754378318787, 0.0020186363253742456, -1.0957164764404297, -0.11396929621696472, 2.049099922180176, -0.41492578387260437, 0.9298009872436523, -1.0312168598175049, 0.10533149540424347, -0.2494734525680542, -0.14563637971878052, 0.7782558798789978, 2.1698338985443115, -0.1063050627708435, 1.3247079849243164, 0.3704155683517456, 0.7564952969551086, -0.44869694113731384, -0.6872261762619019, -0.1815694272518158, -0.7536616921424866, 0.009123672731220722, -1.0375372171401978, 0.023145683109760284, 0.19463421404361725, 1.8339016437530518, 1.7419975996017456, 0.5283176302909851, -0.4836968779563904, 0.6267964839935303, 0.8358477354049683, -0.17668025195598602, -0.8293713331222534, -1.771276593208313, -0.01900007575750351, 1.9416769742965698, -0.1866835653781891, 0.4056864082813263, 0.3856768012046814, 0.5051284432411194, 0.7945407032966614, -1.9836710691452026, -0.12875913083553314, 0.3988375961780548, -0.06705044955015182, 1.2387086153030396, -0.3480604887008667, 0.9469493627548218, 1.7920414209365845, -0.2254084050655365, 0.057759691029787064, -0.2623107433319092, 0.7974582314491272, -2.1079933643341064, -1.4305156469345093], [-0.2600786089897156, 0.9074793457984924, 0.6277872323989868, -0.7818407416343689, 1.6447612047195435, 0.1508508175611496, -0.585057258605957, 0.5723842978477478, 1.3474761247634888, -0.024584978818893433, -1.475991129875183, -0.5078163146972656, 0.13803061842918396, 0.9861943125724792, -0.958077609539032, 0.30879470705986023, -1.1394537687301636, 0.12330487370491028, -1.2868380546569824, -0.41652217507362366, -1.4775577783584595, -0.9329387545585632, -1.544848918914795, -1.9235820770263672, 1.2739397287368774, 1.228738784790039, 0.5300590991973877, -0.30698293447494507, 1.5223150253295898, -0.229792520403862, 0.4800225496292114, 1.1913615465164185, 0.279014527797699, 1.5368289947509766, -0.08661879599094391, -0.09483730792999268, 0.19175629317760468, -1.0959360599517822, -1.0146629810333252, 0.8395438194274902, -1.6665738821029663, 1.1950830221176147, 0.33906376361846924, 1.2519441843032837, 0.8313003182411194, 0.668026864528656, 1.8131299018859863, -0.0066193523816764355, -0.03390422835946083, 1.1996455192565918, -0.5904881954193115, 1.2064158916473389, 0.7611380219459534, -0.43198567628860474, -1.2055048942565918, -1.955995798110962, 1.9460760354995728, -1.4253811836242676, -1.141318440437317, 0.9324416518211365, 0.5389015078544617, 0.2745162844657898, 0.14609888195991516, -0.3657054603099823], [-0.8646954298019409, -0.6852208971977234, 0.6529102921485901, 1.1334609985351562, 1.2387707233428955, 0.014396779239177704, -2.2000515460968018, 0.5596252679824829, -0.0993688553571701, -2.6409964561462402, 1.4376649856567383, 0.2722074091434479, -0.0033549123909324408, 0.24939411878585815, 0.1734665185213089, 0.2923654615879059, 0.17090201377868652, 0.07970554381608963, 0.630560576915741, 0.4072439968585968, 2.48055362701416, 0.20415771007537842, -0.44723913073539734, -0.5324724316596985, 0.45582911372184753, 0.19513754546642303, 0.322083443403244, -0.12491130083799362, -0.10494505614042282, 0.833539605140686, 1.4451780319213867, 0.42491820454597473, -0.02818743884563446, -1.084568738937378, -1.0856214761734009, 0.12048393487930298, -0.17535515129566193, -0.029498199000954628, -1.0472146272659302, -0.06253772974014282, -1.8932443857192993, -1.1398168802261353, -0.7421925663948059, 0.6393010020256042, -0.5662417411804199, -0.9038388729095459, 0.3102126121520996, -0.06312230974435806, 0.957832932472229, 0.3540431261062622, 0.24079760909080505, -0.16631868481636047, 0.6837397813796997, -0.7159861922264099, -1.5155874490737915, 0.9188432097434998, -0.4650467038154602, 0.05570192635059357, 0.6328984498977661, 1.02590012550354, -2.90317702293396, -1.0468684434890747, 1.2337498664855957, -1.7055178880691528], [0.47626402974128723, -0.3627007007598877, -0.8921003341674805, -0.1311357170343399, -1.6965107917785645, 0.40589582920074463, 1.1765064001083374, -1.4324692487716675, -0.6737122535705566, 1.201277494430542, 0.3682346045970917, -0.3279290795326233, 1.4351848363876343, -1.6871209144592285, -0.2809986174106598, 0.08910386264324188, -0.3242110311985016, 0.7949813604354858, -1.6841487884521484, 0.0419415645301342, -0.7951081395149231, 1.1987022161483765, 1.0446423292160034, -0.7540777921676636, -1.110982060432434, -0.05243905261158943, -0.86948162317276, -0.5787657499313354, 0.42370519042015076, -0.4685525596141815, 1.8138983249664307, 0.059520427137613297, -0.5323241949081421, -1.1301554441452026, -0.5311712622642517, 1.1020461320877075, 0.5418751239776611, 1.4253020286560059, 1.3063513040542603, 0.5486820340156555, -0.7083648443222046, -0.16049906611442566, 0.0593281127512455, -1.6690553426742554, 0.3216547667980194, -1.0518375635147095, -0.09585031121969223, -0.5272992253303528, 0.5736538171768188, 2.6331241130828857, -1.2069087028503418, -0.027186580002307892, 2.0459628105163574, -0.938170850276947, -0.2334383726119995, -0.11366590857505798, 0.30666616559028625, -1.6295677423477173, 0.3029867112636566, 0.60346519947052, 2.554124355316162, -0.08022479712963104, -0.5807622075080872, 0.13223762810230255], [-2.6241302490234375, 0.33901074528694153, 1.565286636352539, -0.03310404345393181, -0.03400510922074318, -0.24262818694114685, 0.14173579216003418, -1.2457361221313477, 1.556157112121582, -0.45594120025634766, -0.4500073790550232, 1.6717227697372437, 0.03138326480984688, -0.41074836254119873, 1.6706821918487549, 0.19934625923633575, 0.9619471430778503, -0.5526202321052551, -1.7544792890548706, 0.19771327078342438, 1.4599751234054565, -0.43941211700439453, 0.3906843364238739, 0.7519165873527527, 0.29140812158584595, 0.10033553838729858, 0.8053041100502014, 0.2440894991159439, -0.5051175951957703, 1.0927962064743042, 0.8403662443161011, 0.41254448890686035, 0.7595425248146057, -2.6139895915985107, 0.305851548910141, 0.07747472077608109, -1.9600838422775269, 2.1217827796936035, 0.3910723030567169, -0.4022234380245209, -0.33886682987213135, 0.3079967498779297, 0.024515388533473015, -1.5770111083984375, -0.3604246973991394, 0.49143627285957336, 0.6596852540969849, 0.913526177406311, -1.0220931768417358, -0.5900863409042358, 0.4200451970100403, -0.5641858577728271, 0.26408806443214417, -0.7852447628974915, 0.6409056186676025, 0.11563713848590851, 0.6402919292449951, 0.09382452815771103, -0.7299577593803406, -1.164474606513977, 0.10823257267475128, -0.5488094091415405, -1.4282312393188477, -0.483622670173645], [0.8639504313468933, 1.3144090175628662, 0.8382378220558167, 1.4408756494522095, -0.29636409878730774, -0.07282564043998718, -0.8781378865242004, -0.6975635290145874, 1.6998779773712158, 0.5192322731018066, 0.7358409762382507, 2.7045629024505615, 2.023057222366333, -0.7820925116539001, 1.0720834732055664, -0.6707949638366699, 1.0838983058929443, -1.800173044204712, -0.8540792465209961, -0.8314402103424072, 2.106879711151123, -1.6960121393203735, 1.2090988159179688, 1.1745225191116333, 0.21123374998569489, 0.9305042028427124, 2.6903953552246094, 0.3472551703453064, -0.1859285682439804, 1.4567855596542358, 0.8817878365516663, -1.0671945810317993, 0.28148236870765686, 0.37875786423683167, -1.0624080896377563, -0.8689408898353577, -0.5779282450675964, -1.1256519556045532, 0.6099210977554321, 0.3916228711605072, -2.1584393978118896, -0.34197917580604553, 0.24037520587444305, -1.2846587896347046, -0.2018256038427353, 0.3723554313182831, -1.6439969539642334, -0.23576058447360992, -2.324458599090576, 0.8270865082740784, -0.6788032054901123, -1.4082980155944824, -1.813498616218567, -1.6186257600784302, 1.3463342189788818, -0.8091784119606018, -0.5387489199638367, -0.36283519864082336, -1.4947246313095093, 0.8040604591369629, -2.514575481414795, 0.9299538731575012, -0.7476893067359924, 0.062265776097774506], [0.31638234853744507, 0.864777147769928, -1.245194435119629, 0.6316952705383301, -0.042534977197647095, -0.1497233510017395, -0.279050350189209, 1.1402994394302368, 0.4842745363712311, 1.2972815036773682, 0.9357013702392578, -2.0091545581817627, -0.10172495245933533, 1.0270110368728638, -0.9157797694206238, -1.709349513053894, -2.3339133262634277, 2.0197603702545166, -0.9423668384552002, 0.16915524005889893, -1.0996613502502441, 0.4261535108089447, 1.680699348449707, -1.4839746952056885, -0.07135990262031555, -0.7865275144577026, -1.9219657182693481, -0.9180206656455994, 0.022351490333676338, 1.2539845705032349, 1.299833059310913, -0.02115975320339203, -1.6937263011932373, 0.6718358397483826, -0.008554019033908844, -0.01956520415842533, -1.1728698015213013, -0.162626251578331, 2.0049469470977783, 1.5216397047042847, 1.301401972770691, 0.6831285357475281, -0.40635454654693604, 0.0019021439366042614, 0.1545584350824356, 0.3991532623767853, -0.3562451899051666, -0.8836852312088013, 1.1446409225463867, -0.38013988733291626, 0.8623542189598083, -0.17686472833156586, 0.695436954498291, 0.5290508270263672, 0.760155200958252, 0.7070104479789734, -0.9656563997268677, -1.3841845989227295, -0.04374552145600319, 1.6943448781967163, -0.8474162817001343, -0.9452128410339355, 2.3303966522216797, -1.5767143964767456], [-0.5816718935966492, -0.41499564051628113, -0.6903476715087891, -0.6569643616676331, 1.1038422584533691, 0.2124847173690796, 0.18789586424827576, -1.6894078254699707, 0.8133772015571594, 0.1701420545578003, 0.15670719742774963, 0.8441293239593506, 0.3222537636756897, -0.026573948562145233, 0.4295389950275421, -0.7183917760848999, -0.9197143316268921, -1.3769526481628418, 0.9888535737991333, -0.6619219779968262, 0.18392722308635712, 1.1356672048568726, 1.847802996635437, -0.3610532581806183, 0.1748092919588089, 0.10296069085597992, -1.32536780834198, -0.4818105101585388, 2.8531901836395264, -0.057821985334157944, 1.8103140592575073, 0.784964382648468, 1.798912763595581, 1.359023928642273, 0.23863103985786438, -1.2540137767791748, 1.4669009447097778, 0.7286076545715332, 0.5011237859725952, 0.8914228677749634, -0.801724374294281, 0.1388131082057953, 0.002337648067623377, -1.5085853338241577, -1.217527985572815, -0.12283164262771606, -2.2543201446533203, 0.753483235836029, 0.7880116105079651, -0.0745398998260498, 0.2870529890060425, -1.8507272005081177, 0.17429620027542114, 0.7635998129844666, 0.5131475329399109, 0.9346216320991516, -0.06840269267559052, 2.0264604091644287, 0.43739089369773865, 0.8038463592529297, 1.0973892211914062, -0.7842851281166077, -1.14949631690979, 0.790923535823822]], [[-1.256474494934082, 0.07736048102378845, -0.6027345061302185, -0.9821063876152039, 0.9222771525382996, -0.6194740533828735, 1.0727521181106567, -0.09689264744520187, 0.5969699025154114, 0.32795432209968567, -0.4161047339439392, 0.0652676597237587, 0.6484112739562988, -1.1996281147003174, 0.6564721465110779, -1.2158125638961792, -0.008612111210823059, -1.0739315748214722, 0.9835004806518555, 0.21167947351932526, 0.2746770977973938, 1.756014347076416, -1.2474831342697144, -0.05324898287653923, 1.448454737663269, -1.5204936265945435, 0.9716125726699829, 0.8052693605422974, 1.0382273197174072, -0.8313061594963074, -0.07689738273620605, -1.7105482816696167, 0.003470249241217971, -1.5629953145980835, -0.6972870826721191, 0.5603207349777222, -0.4006485641002655, -1.760683536529541, -0.019914474338293076, 2.3259084224700928, 0.3398369252681732, 0.6846471428871155, -0.3849954605102539, 0.7178567051887512, 0.3988131582736969, 0.2513168156147003, 0.24449530243873596, -0.03218535706400871, -0.231904074549675, 0.017980191856622696, -0.9276303648948669, 0.249210387468338, 0.12176751345396042, 1.5521655082702637, -0.8411056995391846, -1.8283753395080566, 2.134498357772827, -0.1205352246761322, 0.09174773097038269, -1.1921781301498413, 0.3241455852985382, 0.05617665871977806, 0.08089356124401093, -1.4574646949768066], [0.4628725051879883, 0.6063375473022461, -0.9331610202789307, -0.5591902136802673, -0.13281725347042084, -0.30639317631721497, -0.2832806706428528, 0.8107817769050598, 1.165527105331421, -1.8971138000488281, 1.323367714881897, 0.1668599396944046, 0.3179636001586914, 0.3290240466594696, 0.9359439611434937, 0.6962593793869019, -1.5688550472259521, 0.8707560896873474, -1.1553701162338257, 0.3822036683559418, 0.21011683344841003, -1.5739116668701172, 0.00763573357835412, -0.2656339704990387, -0.688567578792572, 2.055619239807129, 0.1570737510919571, -0.38075757026672363, 1.765931248664856, -0.035051796585321426, -0.14131540060043335, -2.0396265983581543, 0.6059733033180237, 1.0125623941421509, 1.936902403831482, -1.0317901372909546, 1.0567675828933716, 1.4211300611495972, 0.7312470078468323, 0.0004376191063784063, -1.6787775754928589, 0.1879522204399109, -0.5667224526405334, -1.7716076374053955, 0.6081501841545105, -0.12629400193691254, -0.24238882958889008, 1.999988079071045, -0.5465682744979858, -2.113917350769043, -2.260662317276001, 0.5262123346328735, -0.2282325029373169, -0.7765992283821106, -2.1633615493774414, -0.8526902794837952, -1.1533225774765015, -0.12701547145843506, -0.8885976076126099, -0.9055554866790771, -0.06567306071519852, -0.8213518857955933, 0.30681097507476807, -3.4890921115875244], [-1.283579707145691, 1.2037767171859741, 0.1792767196893692, -1.1309115886688232, -0.219748392701149, -0.6168520450592041, -0.4062117338180542, 1.4187238216400146, -0.4551678001880646, 0.17878110706806183, -0.8977140784263611, -0.788196325302124, -1.6043168306350708, -0.7006797194480896, -0.4537413716316223, -0.3494471311569214, 0.01974780112504959, 0.061055779457092285, 0.6308714747428894, -0.2514100968837738, 1.2957452535629272, 0.5811610817909241, 1.3666226863861084, -0.08464314788579941, 1.1991623640060425, 0.06524880975484848, -1.7045265436172485, -0.05154397338628769, -0.20740807056427002, -1.5042614936828613, -1.028358817100525, -0.6435770988464355, 2.279287099838257, 0.28586164116859436, 1.8088774681091309, 0.9187138080596924, 1.070163369178772, 0.797220766544342, -1.5280883312225342, 1.2445708513259888, -1.035203218460083, 0.42310360074043274, -0.12211237847805023, 1.6191723346710205, 1.192905068397522, 1.8358203172683716, 0.4801208972930908, -0.00454776082187891, 0.7659798264503479, 2.4078879356384277, 0.5153617858886719, 0.03587067499756813, -1.2473448514938354, -0.760235071182251, -0.09207724034786224, 0.6738526821136475, 0.620242714881897, -0.1053115651011467, 0.12852054834365845, 1.5394366979599, 1.8534750938415527, 1.2020151615142822, 1.5357857942581177, 1.2501819133758545], [1.8346267938613892, -1.266554355621338, -0.5137951374053955, 0.23736003041267395, 0.39106374979019165, 1.3472763299942017, -1.8483145236968994, 0.15010730922222137, 0.269502192735672, -0.6765498518943787, -0.13676701486110687, 1.0100003480911255, 0.47231218218803406, -0.12119347602128983, 0.2133304625749588, -0.11790558695793152, 0.9489354491233826, -0.09370438754558563, 0.9949321746826172, -0.5499192476272583, -0.023564215749502182, -1.11411452293396, -0.19425441324710846, 0.8555719256401062, -1.0062843561172485, -0.33976125717163086, 0.11938902735710144, -0.3426598906517029, 0.8570409417152405, -1.2319366931915283, 0.8793560266494751, 1.344630241394043, 1.6850330829620361, -0.046261727809906006, -0.7885118722915649, -1.566705584526062, 0.7056165933609009, 0.8463314771652222, 0.31323671340942383, 0.3673610985279083, 0.6974493861198425, -0.341010183095932, -1.45907461643219, -1.4159059524536133, -0.7706493139266968, -0.9017643332481384, -1.5783023834228516, -0.41286250948905945, -1.0220776796340942, 0.6832398772239685, -0.05107981339097023, 0.8219492435455322, -1.311927318572998, -1.0842654705047607, -1.5840054750442505, 0.3100616931915283, 3.1597750186920166, 0.24542008340358734, -0.4130656123161316, 0.23803766071796417, -0.007701708935201168, 0.9950780272483826, 0.01981261931359768, -0.8630015254020691], [-1.8907221555709839, 0.38540878891944885, 1.0339497327804565, 0.17416469752788544, -0.8695354461669922, -0.41264787316322327, -0.30134317278862, 0.609575629234314, -0.0373137928545475, 1.2036628723144531, -0.1780203878879547, -0.6263366341590881, 1.8862498998641968, -0.983154296875, -0.7684646248817444, 2.1231918334960938, 0.3077166676521301, 1.8227672576904297, -2.4914472103118896, 1.2892688512802124, 0.024232281371951103, 0.1291380077600479, 0.5211510062217712, -2.7540884017944336, -0.880375325679779, 0.4748156666755676, 3.280484437942505, -0.734463095664978, 2.4022715091705322, 0.6666901707649231, 1.0606590509414673, -0.9445372223854065, 0.6408244371414185, -0.8412412405014038, -1.19634211063385, 1.3122743368148804, -0.6845633387565613, -0.4900459945201874, 0.7229471802711487, -0.7951647043228149, -1.2512199878692627, 0.25068992376327515, 0.29657167196273804, 1.5548866987228394, -1.1132643222808838, 0.7431970238685608, 0.741662859916687, -0.4214533269405365, -0.14950981736183167, 0.04107408598065376, -1.0215051174163818, -1.315905213356018, 0.7301647663116455, -1.0596551895141602, -0.9686294794082642, 0.8134691715240479, -0.534144401550293, -0.3964405953884125, -1.4020283222198486, -1.2373522520065308, -1.3910726308822632, -1.3860958814620972, 1.258571743965149, -0.9760587811470032], [-2.3431038856506348, -1.1263753175735474, 0.6967557072639465, -1.1906239986419678, -0.0696740373969078, -0.9768961071968079, -1.8019131422042847, -0.5871549248695374, -1.0703202486038208, 0.13781099021434784, 0.6738025546073914, -0.10938538610935211, 2.619720220565796, -1.110555648803711, 1.3088946342468262, 0.38069814443588257, 1.367080569267273, 0.49181103706359863, -2.3132083415985107, -0.3493176996707916, 0.9929517507553101, -0.02758386917412281, -1.3802120685577393, -0.6042711734771729, 0.2782595455646515, 1.1835925579071045, -0.662032961845398, 0.04236043244600296, -1.0001064538955688, -1.2735131978988647, -0.5881301164627075, 1.2959563732147217, 0.6613765954971313, -0.3469172716140747, -0.09621936827898026, 0.46980953216552734, -0.19815851747989655, -0.5142219662666321, -0.38081440329551697, 0.5668272376060486, -0.4785928726196289, 0.7067695260047913, -1.961521863937378, -0.854141354560852, -0.5196807384490967, 0.5035271048545837, -1.021882176399231, -0.6350343227386475, 0.22382725775241852, -0.20150126516819, -0.7761430144309998, -0.4345782399177551, -1.0312949419021606, 0.7105249762535095, 0.3398021459579468, 0.7663951516151428, 2.173912763595581, 1.1747143268585205, 0.4217837452888489, -0.16563311219215393, 2.218447208404541, 0.19466865062713623, -0.25797784328460693, 1.2692523002624512], [0.4117867648601532, -0.26381590962409973, 0.44421863555908203, 1.9802014827728271, 0.166892871260643, 0.41853103041648865, -0.8326524496078491, -0.41872110962867737, 0.9571453928947449, -0.021325645968317986, 0.19744333624839783, 1.183922529220581, -0.2429741770029068, 1.6476994752883911, -0.555851399898529, -0.6657913327217102, -0.7138451337814331, -1.4098362922668457, 0.6448240876197815, 0.11049598455429077, 1.4621315002441406, -0.6484695076942444, 0.8371844291687012, 1.4197953939437866, -0.14677178859710693, -0.09642885625362396, -2.4290921688079834, 2.00850510597229, -0.06405671685934067, 0.7521620988845825, -0.46113643050193787, 0.4048464596271515, -1.095330834388733, 0.807016909122467, 0.15720871090888977, -0.31606048345565796, 0.48843416571617126, -1.7440528869628906, 0.21442176401615143, -0.3899080455303192, 1.4976296424865723, 0.8168826103210449, -0.14408200979232788, -0.25363150238990784, 3.027841091156006, -0.9625516533851624, 1.9403542280197144, 0.9593336582183838, 0.8167067766189575, -1.8807340860366821, 0.861061692237854, -0.4763568937778473, -0.8841438889503479, 0.6219140291213989, -0.6908262968063354, 0.5945615768432617, -0.9252094626426697, 0.6085829734802246, 0.4558362066745758, -0.8358479142189026, -0.9486767053604126, -0.6734093427658081, -0.5301512479782104, -0.6025046110153198], [1.852903127670288, 0.4144555926322937, -0.3521604537963867, -0.3363032341003418, -0.8189142346382141, 0.2642752230167389, -0.07895781099796295, -0.6958801746368408, -0.7114008069038391, -0.4240822196006775, -1.077291488647461, 0.09817306697368622, -0.930112361907959, 0.5227359533309937, -0.3476731777191162, 0.5173784494400024, -0.5966096520423889, -1.88602876663208, -1.3306903839111328, -0.30026423931121826, 0.3775738775730133, 0.40221843123435974, 0.5155473351478577, 0.6871378421783447, 0.36985957622528076, 0.4444774091243744, -0.21929647028446198, 0.5644184350967407, -0.10225122421979904, -0.06550837308168411, 0.1972229778766632, 2.3073863983154297, -1.4591786861419678, -0.46659308671951294, -0.25178998708724976, 0.4394550919532776, 0.030403653159737587, 1.6543138027191162, -1.5394114255905151, 1.8360337018966675, -1.2312308549880981, -0.6932327747344971, 0.5829170346260071, 0.10989007353782654, 0.0028024918865412474, 0.1010860726237297, 0.41663408279418945, 0.990801215171814, 0.4378989636898041, -1.2280985116958618, -0.7072792053222656, 1.4000340700149536, 1.9535826444625854, 0.9641135931015015, 1.0990877151489258, 0.3367008864879608, 2.1004538536071777, -0.5456416606903076, -0.30883529782295227, -0.7497166395187378, 1.592954397201538, -0.49618202447891235, 0.5355872511863708, -1.3538793325424194], [-1.4378212690353394, 1.0183064937591553, 0.7379837036132812, 0.878753125667572, 0.23666289448738098, -0.2587861716747284, -0.3739846348762512, -1.0723695755004883, -0.34636470675468445, 1.3794361352920532, -0.12160094082355499, -1.2731781005859375, -0.20390133559703827, -0.9462082386016846, -1.1088943481445312, -0.07637085020542145, -0.7296434640884399, -1.2250570058822632, 1.5490106344223022, 1.7468217611312866, 0.17093303799629211, 1.5040730237960815, -0.4287012815475464, -0.0072801001369953156, 1.3300824165344238, 0.3224194347858429, 0.8401212096214294, -2.7249093055725098, 0.3085741102695465, -0.4050181806087494, -0.09118630737066269, -1.5000832080841064, -0.7580959796905518, 0.6101497411727905, 0.5960333347320557, 0.7825926542282104, 0.85992032289505, 1.1054673194885254, 0.3939518630504608, -0.22872893512248993, -0.5660899877548218, -0.41188210248947144, -0.9606953263282776, -0.9880048036575317, -0.08396982401609421, -2.273559093475342, -1.2232555150985718, -0.7990161180496216, 0.5299466848373413, 1.0802807807922363, 0.33431708812713623, -0.03870897367596626, -0.48555073142051697, 1.118587613105774, 0.7146350145339966, -0.6497265696525574, -0.017248759046196938, 0.4223219156265259, 0.9239057898521423, -0.3174450993537903, 1.5150575637817383, 1.1879867315292358, -0.40283674001693726, -0.17259639501571655], [-0.8989632725715637, -0.8745676279067993, 0.34387657046318054, 1.4575897455215454, 1.4731487035751343, 0.2334730476140976, -0.32460805773735046, -0.594416618347168, 0.48383259773254395, -0.4040856659412384, 0.772034227848053, -0.05897550657391548, 0.355744332075119, -0.45271843671798706, 2.0842161178588867, 1.2032252550125122, 0.15742062032222748, -0.8073006272315979, 0.633104145526886, 0.824955940246582, -1.0576523542404175, 0.14946265518665314, 2.1114773750305176, 0.2559845745563507, -0.3155668377876282, 0.6699381470680237, 1.8983629941940308, -0.35976630449295044, -0.3484162986278534, -0.1523588001728058, 0.16521680355072021, 1.2786155939102173, 0.42656999826431274, -2.1184041500091553, 0.5315694212913513, 0.4979761242866516, 0.7367697358131409, 0.37691470980644226, -0.020146511495113373, 0.6139426231384277, -0.5744167566299438, -1.5768693685531616, -0.2356051802635193, -0.9987313151359558, 1.7751117944717407, -0.02259250357747078, 1.6441134214401245, -1.2590882778167725, -1.1267911195755005, 0.5007064342498779, -1.6315810680389404, -0.5709758400917053, -1.868010401725769, 0.4814133644104004, -1.660117745399475, -0.08186803758144379, -1.711890697479248, 0.5296099781990051, 1.1102181673049927, 0.16424964368343353, -0.9433190822601318, 0.46220409870147705, 1.0122227668762207, 0.7989479303359985], [1.2946593761444092, 1.9302805662155151, 0.2597943842411041, -0.22692182660102844, -0.6023266315460205, 0.018776608631014824, -0.20093780755996704, -0.499337375164032, 0.36964184045791626, 2.00644850730896, -0.7636719346046448, -0.45962753891944885, -0.331974059343338, -1.0034511089324951, 1.0679951906204224, -0.13810276985168457, 0.6084507703781128, -1.5930707454681396, 0.5707660913467407, -0.11970788985490799, -1.4413307905197144, 0.8660524487495422, 0.9487757086753845, 0.2321598380804062, 0.47581371665000916, -0.9398449659347534, 1.0351611375808716, -0.22718358039855957, 0.48386818170547485, 0.6052852272987366, 0.4902229905128479, -0.7192102670669556, 0.28369516134262085, 0.15395978093147278, 1.1887308359146118, -0.7401849627494812, 0.9670329689979553, 0.4880593419075012, 0.0675697848200798, 1.8977575302124023, 0.8222788572311401, -0.5326683521270752, 0.3140498995780945, -0.7374113202095032, -1.153651237487793, -0.48984450101852417, -0.4775482416152954, 1.7176231145858765, 0.810215175151825, -2.4088127613067627, -0.14535285532474518, -0.6888182759284973, -0.13287459313869476, 0.1854962557554245, 0.04844951257109642, 0.4300859868526459, -0.7742994427680969, 1.4190995693206787, -0.12620404362678528, 1.237034559249878, -0.09160531312227249, 0.5015392899513245, -0.9523482322692871, 1.1138787269592285], [0.29490765929222107, -0.39645102620124817, 0.38532671332359314, 0.4753826856613159, -1.5354543924331665, -0.7465856075286865, -0.9922381639480591, 0.8310493230819702, 0.18762171268463135, -0.6644507050514221, -0.7077614665031433, -0.039350010454654694, -2.2990148067474365, -0.16877107322216034, 1.5640983581542969, -1.4035494327545166, -1.1968687772750854, -1.1314923763275146, 0.4406205415725708, -0.7534507513046265, 1.870538592338562, -0.20240260660648346, -0.49794867634773254, -1.1012840270996094, 0.6805511116981506, 1.071608543395996, 0.17080950736999512, 1.3044053316116333, 1.242796540260315, 0.3613174557685852, 0.06728771328926086, -0.5427314639091492, 0.22998665273189545, -0.33789461851119995, 1.489331841468811, 0.767981767654419, -0.15290775895118713, -0.3733164966106415, 0.43045011162757874, 1.0660746097564697, 0.30078452825546265, 1.0407698154449463, -0.42374515533447266, 0.2492169290781021, 0.678574800491333, 0.2271493375301361, 0.4186370074748993, 1.0504661798477173, 0.03492581099271774, 1.037218451499939, -0.4703746438026428, 0.5396657586097717, -0.3772940933704376, -0.32690781354904175, 0.655674934387207, 1.3336074352264404, -0.058609090745449066, 0.16296041011810303, 1.394927978515625, -0.7243450284004211, 0.9893326759338379, -1.0811567306518555, -1.8002487421035767, 0.4312465190887451], [-0.20289508998394012, -0.42567312717437744, -2.3121469020843506, 0.4457888603210449, -0.38042548298835754, -1.0975983142852783, -0.03929280489683151, -1.119173288345337, -0.7006880640983582, -0.743288516998291, 1.036751627922058, -0.0502912774682045, 1.3461846113204956, -0.7511456608772278, -1.1358426809310913, 1.8109080791473389, -1.326167106628418, 1.2654954195022583, -1.0491005182266235, 0.7671260833740234, -0.6859959959983826, 1.5593405961990356, 2.025953769683838, 1.0971591472625732, 0.4107394814491272, -0.30582094192504883, 0.8485366106033325, 2.1631383895874023, -0.38581642508506775, -1.0673532485961914, 1.855400562286377, -0.15129543840885162, 0.8064190745353699, 0.2669338285923004, 0.34848177433013916, -0.1976155787706375, -0.056623492389917374, -1.589194893836975, -0.4844615161418915, -0.7823276519775391, 1.2475695610046387, -0.22885139286518097, 0.19063137471675873, -1.5898244380950928, 0.15996578335762024, 1.609471321105957, 0.2534945607185364, 0.9643060564994812, -0.22611163556575775, -0.5263950824737549, 1.046466588973999, -0.3636203408241272, -0.9595946073532104, -0.015127908438444138, -0.29786303639411926, 1.1378110647201538, 0.8100088834762573, -1.3530939817428589, 0.596145510673523, -1.2496598958969116, 0.2651059925556183, 0.6556275486946106, -0.4312736392021179, -0.4396377205848694], [0.8494310975074768, -0.21529564261436462, -1.0392838716506958, 1.7186962366104126, 0.9644219279289246, 0.5621758699417114, 0.4356314241886139, 1.1098506450653076, 0.17943821847438812, 0.8839022517204285, 0.42547059059143066, -0.5741647481918335, 1.1636534929275513, 0.2796519100666046, -0.6976014971733093, -0.3311852514743805, -2.2853822708129883, -1.5536330938339233, -1.348146915435791, 0.7956990599632263, 0.762464165687561, 1.2402788400650024, 0.5089648365974426, 1.8096579313278198, -0.37223732471466064, -0.6800718307495117, -0.8567442297935486, 0.26151347160339355, -0.5864918231964111, -0.8570237755775452, -0.4819689989089966, 0.20421355962753296, -1.241112232208252, 0.03054048679769039, 0.018350427970290184, 1.7065799236297607, 0.18260003626346588, -1.2554631233215332, 0.28380337357521057, 0.2198420614004135, -1.2326818704605103, -3.0200252532958984, 0.7750603556632996, -0.16169552505016327, -0.6993016600608826, -0.15083551406860352, 1.3017535209655762, 1.7662708759307861, -0.4445127844810486, 1.545525074005127, -0.029086990281939507, 0.02635383978486061, -0.07475262880325317, -0.04334151744842529, -0.12689724564552307, -0.6685713529586792, 0.03407662361860275, -0.9958080649375916, 0.577201247215271, -0.6321569681167603, 0.5038849115371704, -0.10716787725687027, -1.7749274969100952, 1.1907894611358643], [0.30621784925460815, -0.05482300743460655, -0.32525354623794556, 1.8049708604812622, -0.23402263224124908, 0.9681760668754578, 0.21120670437812805, -1.3210361003875732, 0.728491485118866, -1.5151286125183105, 0.4537009000778198, 0.41374462842941284, 0.2375398874282837, -0.7435579299926758, 1.397802472114563, 1.7701517343521118, -0.9155606031417847, -0.38045865297317505, 1.6291191577911377, -0.4323292672634125, 0.961184561252594, -0.9784068465232849, -0.633400559425354, 0.26013097167015076, 0.45214128494262695, -1.4457045793533325, -1.0499062538146973, 0.5112688541412354, 1.2757476568222046, 0.5147407054901123, 0.35967937111854553, 0.43580010533332825, -1.2190502882003784, 0.3962579369544983, 0.8991712927818298, 0.5954474210739136, -1.166929006576538, -0.958499014377594, 1.1200820207595825, -1.384940266609192, -1.0681498050689697, 1.583078384399414, -0.9018523693084717, -0.7317481637001038, 0.6386840343475342, 0.7848655581474304, -0.16740551590919495, -1.4323614835739136, -0.5178028345108032, 0.16304154694080353, -0.9197327494621277, -0.9756007194519043, 1.2500271797180176, 0.5410477519035339, 0.7814403176307678, 0.8961166143417358, -1.0051637887954712, 0.4836586117744446, 1.1929501295089722, 0.679314911365509, -1.1566542387008667, -1.5547057390213013, -0.803153395652771, 0.5467050075531006], [-0.18544551730155945, 0.278155654668808, -1.2629302740097046, 0.8317814469337463, -0.7319740653038025, 1.1288713216781616, -0.02692096121609211, -2.983304738998413, -0.20099715888500214, -0.8784583210945129, -0.8480167388916016, -0.16243065893650055, 1.0687962770462036, 0.843956470489502, 1.3950250148773193, -0.006775322370231152, -1.8736416101455688, -0.5627657771110535, -0.9102773070335388, -0.6916677355766296, 2.660832643508911, 0.7955988645553589, -0.9716129899024963, 1.1903032064437866, -0.2986043393611908, -1.3516161441802979, 1.168671727180481, -0.479875773191452, 0.12517546117305756, 2.297947645187378, -1.0188980102539062, 1.0360150337219238, -0.31614920496940613, -1.1174938678741455, -0.25544971227645874, 0.17025402188301086, 0.18396884202957153, -1.5423492193222046, 0.8352594971656799, -0.8572018146514893, 0.3042236566543579, 0.20090848207473755, -0.5590395331382751, 0.5946524143218994, -0.7217796444892883, -0.949777364730835, -1.4248555898666382, -1.867972493171692, -0.7026321887969971, -2.198394775390625, -0.3678354322910309, -0.5476751327514648, 1.6355128288269043, -1.1710472106933594, 1.1364130973815918, 0.3579765260219574, 0.28504258394241333, -0.6431961059570312, -0.8386884927749634, -0.33704936504364014, -0.0642470046877861, 0.5602267384529114, -0.5295238494873047, 1.5474653244018555]], [[0.2376844435930252, -1.0341466665267944, -1.1783154010772705, -1.376650094985962, -0.3012644946575165, 0.1899993121623993, 0.39792871475219727, 1.2573450803756714, -0.6109111905097961, 1.263148546218872, -0.20879870653152466, -1.2961629629135132, 0.5655114650726318, 1.6571152210235596, -0.3604848086833954, 1.5028095245361328, -0.13337182998657227, -0.08298732340335846, -0.7100608944892883, -1.8428410291671753, -0.19885297119617462, -0.1772790402173996, 2.2174551486968994, -1.4922338724136353, -0.5297401547431946, 0.2872321605682373, -0.2231827676296234, -1.1552789211273193, -1.3329817056655884, 1.1977821588516235, -1.2902368307113647, -0.09040362387895584, -0.5368956923484802, 0.1963765174150467, -0.029086623340845108, 0.6171804070472717, 0.1898716241121292, 3.531663179397583, -0.5755002498626709, -0.6307498216629028, 1.1526025533676147, 1.5878263711929321, 0.7764789462089539, -0.8319397568702698, 2.6008217334747314, -0.42470046877861023, -1.1448540687561035, 0.6872486472129822, -0.01886928081512451, 0.5388840436935425, -1.2328318357467651, 0.838860273361206, 0.12417730689048767, -1.250189185142517, -0.7904442548751831, 0.07810982316732407, -0.5305485725402832, 0.14061017334461212, -0.21894745528697968, -1.4027373790740967, -1.0650700330734253, -0.34562447667121887, 0.10337165743112564, -1.4452776908874512], [0.574055552482605, -0.03750361502170563, -1.276310682296753, -1.4215664863586426, -0.05281725898385048, -1.065730094909668, 0.8345025181770325, 0.9644421339035034, 1.5105040073394775, 0.14101266860961914, 1.9865413904190063, 1.3150159120559692, 0.42888861894607544, 0.80147784948349, -0.5164341330528259, -3.118182897567749, 1.0983068943023682, 0.5442060828208923, 1.736991286277771, 0.5411150455474854, -0.12752681970596313, -0.4405460059642792, -0.5650120377540588, 0.21583804488182068, -1.7724534273147583, -1.0814930200576782, 0.8762532472610474, -2.0076615810394287, -1.5287182331085205, -0.5604720711708069, -0.37819704413414, 0.09841744601726532, 0.8811007142066956, 0.31697705388069153, 0.623593270778656, -0.559839129447937, 0.051346153020858765, -0.6860194206237793, -0.08713014423847198, 0.6069824695587158, 0.5255100131034851, -1.87220299243927, -1.1198173761367798, 2.0182058811187744, -2.0275206565856934, 0.08992525190114975, -1.149557113647461, -0.192964568734169, -0.5637987852096558, 0.09119017422199249, -1.088201880455017, 1.069815993309021, 2.21403169631958, -2.4384334087371826, -0.8260675072669983, -1.5027443170547485, 0.14151571691036224, 1.4273366928100586, 1.4152957201004028, 1.3436318635940552, 0.24831610918045044, -0.536372184753418, 1.1526192426681519, -0.592350423336029], [-0.7679307460784912, 0.08941148966550827, -1.220232367515564, -0.16935715079307556, -0.920637845993042, 0.12135239690542221, 0.992047905921936, -0.5344865918159485, 0.4972158968448639, 0.9880134463310242, -0.7551485896110535, -0.049712300300598145, -0.3396786153316498, 1.8394063711166382, -0.4816047251224518, 0.14651602506637573, -1.3879245519638062, -0.9070892930030823, 0.6201841831207275, -0.26400578022003174, 0.7687034606933594, 0.09005006402730942, 2.6467742919921875, -0.21276643872261047, 0.9800443649291992, -0.6944661140441895, -0.4127323031425476, -1.5103859901428223, 2.6688411235809326, -2.077820062637329, 0.14751334488391876, -0.41501426696777344, 0.8284316062927246, -0.5147935152053833, 1.9319367408752441, 1.176098108291626, 0.5752525925636292, 0.5455314517021179, 1.1120167970657349, 0.4912285804748535, 1.659765362739563, 0.9524226188659668, 0.030035853385925293, 0.5216901302337646, 0.2936675250530243, 1.282731056213379, -0.16396719217300415, 0.20250225067138672, -1.1694313287734985, 0.7778228521347046, 0.34858885407447815, -0.5647812485694885, 1.8266050815582275, 0.13034836947917938, -1.0519931316375732, 1.2555075883865356, -0.0755084976553917, -3.0023438930511475, 0.45173877477645874, 0.08945809304714203, 0.18226958811283112, -0.10465089976787567, -0.7131412625312805, 0.22052015364170074], [-1.5977740287780762, -0.08372026681900024, 0.9634135961532593, 1.0978418588638306, -0.40297722816467285, 2.037614583969116, 0.40261754393577576, -0.3468180298805237, 0.25993403792381287, -0.5926457047462463, 0.9643853306770325, -0.8455010056495667, -0.6122591495513916, -1.0585112571716309, -0.7160094380378723, -0.18803143501281738, 1.9595186710357666, 0.33877381682395935, 1.2755241394042969, 0.1449190229177475, 1.0415313243865967, -0.9555771350860596, -1.035093069076538, 1.9607577323913574, 0.4695088565349579, -1.4889787435531616, 0.6967544555664062, -0.10883867740631104, -0.18226486444473267, 0.9165145754814148, -1.1422555446624756, 1.3184292316436768, 0.3490830063819885, -0.39783433079719543, -1.265401840209961, 1.844338297843933, -0.28609588742256165, -1.302280068397522, -0.07707741856575012, 0.5476939082145691, 0.7013795375823975, 1.5576955080032349, 0.636326253414154, 0.01624738797545433, 2.320136785507202, 2.0990424156188965, -1.0546531677246094, 1.0578608512878418, -0.5170668959617615, 1.583986759185791, -0.8697493076324463, 0.38141393661499023, 1.6703126430511475, 0.4210970401763916, 1.1836241483688354, -0.6241598725318909, -1.164780855178833, -1.099281668663025, 0.6985853314399719, -0.5676637291908264, 1.5615769624710083, -0.2562851309776306, 1.438626766204834, 0.43415161967277527], [-1.008139967918396, 0.1386002153158188, 0.5796401500701904, 0.5829049944877625, -0.014744335785508156, 1.2071722745895386, 1.0491958856582642, 1.0417866706848145, -0.6953059434890747, -2.0423152446746826, 0.6714187860488892, -0.7508044838905334, -0.4588521122932434, -0.16163498163223267, 0.4102363586425781, -1.677219033241272, -0.04861164093017578, 1.4559720754623413, -1.3821014165878296, -0.21119609475135803, 1.5442267656326294, 0.9272860884666443, 0.7106500864028931, 0.24335119128227234, -1.3469922542572021, 0.20281638205051422, -0.8722490668296814, 0.21567144989967346, -0.6804844737052917, -1.0944935083389282, 0.08393443375825882, 1.1605384349822998, -0.4103006422519684, 0.27684521675109863, 0.813885509967804, 0.6065249443054199, -1.1890093088150024, -0.4749634265899658, 0.4454396367073059, 0.11049844324588776, 1.8578436374664307, 0.679482102394104, -0.8042291402816772, 0.7697632908821106, 1.2138804197311401, 0.7421479821205139, 0.5988601446151733, -1.3343217372894287, 0.026255056262016296, 0.682756245136261, -0.310396671295166, -0.2561720609664917, 0.5350619554519653, -0.5647339820861816, -0.8186501264572144, -0.07931062579154968, -0.016269981861114502, 0.9288008809089661, -0.0957370474934578, -1.0884650945663452, 0.5529963374137878, -0.7272505760192871, 0.8126926422119141, -0.508187472820282], [0.6492453813552856, 0.3831597566604614, -0.1258544623851776, -1.2019609212875366, -0.3282935619354248, 1.4864698648452759, 0.2596432864665985, 1.2317891120910645, 0.1379862129688263, 0.07829545438289642, -0.9018182754516602, 0.4542944133281708, -0.7724809646606445, -0.7655104994773865, 1.111363410949707, -0.9317512512207031, 2.370243549346924, 0.06538983434438705, -0.49641308188438416, 1.4049818515777588, -0.9127324819564819, 0.1981116086244583, 0.10118179023265839, 1.0183123350143433, 1.798588514328003, 0.4112475514411926, -1.225598692893982, -1.261889934539795, -0.42825639247894287, 0.8730645775794983, 0.1969636082649231, 0.3364540934562683, -0.2508205473423004, 1.4187977313995361, -1.0688376426696777, 1.990577220916748, 1.26265287399292, 0.7657533884048462, 0.039348162710666656, -0.23543761670589447, 0.05939410254359245, -0.48150646686553955, -0.36307594180107117, -0.7521176934242249, 0.8223510980606079, 0.15944361686706543, 0.501099705696106, -0.6747158169746399, -1.1779736280441284, -0.8672982454299927, 0.2819685637950897, -1.1254199743270874, 0.6416962146759033, -0.8955376148223877, -0.4036557972431183, 0.2831224799156189, -1.1471447944641113, -2.0295753479003906, 1.191953420639038, -0.1358472853899002, -0.6067324280738831, 1.6278566122055054, 2.2002360820770264, 0.05231134220957756], [0.341178834438324, 2.657649278640747, -0.5864152908325195, 1.229604959487915, 1.0042905807495117, -0.5409359931945801, -0.9458861947059631, -0.7368704676628113, 0.17119915783405304, 0.37926244735717773, 2.640115737915039, -0.9913631081581116, 0.2300710380077362, -0.7514102458953857, 0.9043951630592346, 0.46724733710289, -1.2744417190551758, -0.4049414098262787, 0.10765296220779419, 0.772471010684967, 0.5813251733779907, 0.7263111472129822, 0.07743563503026962, 0.6570204496383667, 0.13751058280467987, 1.0316225290298462, 0.6429873108863831, -0.8199806809425354, 0.015138978138566017, -0.7021170258522034, -0.05452507361769676, 0.632208526134491, 0.16194570064544678, -0.04425327479839325, -0.5379634499549866, 1.5462924242019653, 0.07825995236635208, -0.8941624760627747, -0.4053542912006378, 1.0155236721038818, 2.0947463512420654, 1.3427225351333618, -0.2598535716533661, 0.9273083806037903, -1.5896728038787842, 0.9028159379959106, -0.1547987461090088, 0.337981641292572, -1.156127691268921, 0.4501473307609558, -1.1998841762542725, -1.5599243640899658, 1.5622144937515259, -1.1675564050674438, 0.9930763840675354, -0.4901351034641266, -0.039744727313518524, 0.06694691628217697, 1.1796844005584717, 0.5012404322624207, -0.8121668100357056, -0.44964227080345154, 1.3879607915878296, 1.4151084423065186], [-0.4072864353656769, 0.9538639187812805, -0.005431880243122578, 1.0714361667633057, -0.9404116868972778, 0.8740315437316895, -0.4989400804042816, -1.2197439670562744, 0.6629244685173035, 0.6253814697265625, -1.4380460977554321, -2.2581748962402344, -0.06952077895402908, 0.4943099915981293, -0.48744890093803406, 0.13446544110774994, -0.3825136721134186, -0.5112877488136292, 0.4277516305446625, 1.2649694681167603, 0.49866095185279846, -0.28250616788864136, -0.6538451910018921, 0.23434822261333466, -2.171501398086548, 0.1375926434993744, -0.08335935324430466, 0.006405659485608339, -0.24392864108085632, -0.17330209910869598, -1.4279048442840576, 0.6775554418563843, -0.29543396830558777, 0.038633182644844055, -0.4739074409008026, -1.4527223110198975, 0.6427724957466125, -0.19842591881752014, -0.23208148777484894, 0.40083184838294983, 1.7148016691207886, -0.01067575067281723, 0.5707886815071106, 1.8718814849853516, -0.27517208456993103, 1.247967004776001, 0.07521532475948334, 1.0412615537643433, 0.15740154683589935, 0.9957903027534485, -0.701547384262085, -0.4429342448711395, -0.573255717754364, -0.25896376371383667, 1.141019344329834, 0.47779518365859985, 0.7084728479385376, -0.16866256296634674, 0.1391945481300354, 0.46999624371528625, 0.408950537443161, -0.5055137276649475, -0.33724498748779297, -1.0063786506652832], [0.8035193681716919, 1.1075878143310547, 1.9465858936309814, -3.022599220275879, 0.8855026960372925, 0.6209810376167297, 2.6916885375976562, -0.40931326150894165, -0.7286598682403564, 0.261883020401001, -1.3990834951400757, -1.3640228509902954, 0.6135121583938599, -1.6320226192474365, -1.0834375619888306, 1.2185560464859009, 0.616369366645813, -1.2684286832809448, 1.4197355508804321, 0.09837165474891663, -0.3274315595626831, 0.16301009058952332, -0.07891504466533661, 1.179919719696045, 0.5907275676727295, -0.8738268613815308, -0.5202476382255554, 0.7822255492210388, -0.6043688654899597, -0.9650158882141113, -2.3792288303375244, 0.8469629287719727, 0.008675879798829556, -0.20740407705307007, -0.8806806802749634, -0.5542364120483398, -0.0361345037817955, -1.1340765953063965, 0.09976045042276382, 1.7121368646621704, -0.5213478803634644, 0.0158638097345829, 1.5823760032653809, 0.10097140073776245, 1.5719095468521118, 1.0225367546081543, 0.9305958151817322, -0.16413745284080505, -0.4539240300655365, 1.333112120628357, 0.14839796721935272, -0.10384184122085571, -2.1861491203308105, -0.13131318986415863, 2.5089852809906006, 0.6596968770027161, 1.8052024841308594, 0.26568523049354553, -1.3633335828781128, 0.18430054187774658, 1.817602515220642, 0.2254113405942917, -0.2435978353023529, -0.1254524439573288], [-1.1273407936096191, -0.30678290128707886, 0.18408356606960297, 1.007245421409607, 0.8182203769683838, -1.0563197135925293, -1.8846975564956665, -0.30578288435935974, -0.7182192206382751, 0.8816930651664734, -0.9771134853363037, 0.6500548720359802, -0.3910331726074219, 1.180495023727417, 0.09764722734689713, -1.9364246129989624, -2.326242208480835, 0.33993229269981384, -0.9981257319450378, 0.9973939657211304, -0.020869823172688484, 1.8069126605987549, -0.01322134118527174, -0.7462223768234253, -2.008134126663208, 0.5002263188362122, -0.21392115950584412, -0.31688588857650757, 2.70031476020813, 0.7976053953170776, -0.9404081106185913, 0.6705132126808167, -1.549302577972412, -0.6488956809043884, 0.8580150604248047, -1.3166965246200562, -0.002242449903860688, 0.06336915493011475, 0.17389526963233948, -0.9803891777992249, -1.4209308624267578, -0.015031237155199051, 0.6358557939529419, -1.0020148754119873, 0.8084143996238708, 1.1043095588684082, 0.32442060112953186, 1.0261847972869873, 0.9863293766975403, 1.216808557510376, 0.27726149559020996, 0.30499300360679626, 0.3611016869544983, 1.2193549871444702, -0.49719586968421936, 1.5071804523468018, -0.9183217883110046, 1.9850945472717285, 0.6710376143455505, 2.741076946258545, 0.2953648865222931, 0.7170491814613342, -1.5355044603347778, -0.24160020053386688], [0.23035173118114471, 0.4512726664543152, 0.7002689838409424, 0.18588511645793915, -0.020422454923391342, 0.986422598361969, -1.4825314283370972, -0.4149706959724426, 0.7296224236488342, -0.04177474230527878, 1.2389990091323853, -0.6432457566261292, -0.3435799777507782, 0.6666651368141174, 1.8195815086364746, -1.2647656202316284, -1.6835395097732544, -1.1772680282592773, -1.2185548543930054, 0.29450681805610657, 0.9712257385253906, 0.01290073711425066, 0.2720201015472412, 0.6628457307815552, 0.41368475556373596, 0.32981154322624207, 0.270978718996048, 1.1760356426239014, -1.0972440242767334, -0.4595486521720886, 0.3020695149898529, 2.744908332824707, -0.8269578814506531, 0.8197419047355652, -0.7017862200737, -0.12611179053783417, -0.7437007427215576, 1.9193602800369263, 0.42006969451904297, -0.459930419921875, 1.0233185291290283, -0.6631523370742798, 0.31333446502685547, -1.2996395826339722, -0.26468712091445923, 0.38684210181236267, 0.384397953748703, -0.8066387176513672, -0.7274262309074402, -1.7075073719024658, -2.084909677505493, -0.19645905494689941, 1.1618103981018066, 0.5110016465187073, 0.6180379390716553, 0.04809848964214325, 0.012796456925570965, -0.0036893223877996206, 1.7159843444824219, -0.29239246249198914, 1.2473583221435547, -1.3711719512939453, -1.0330928564071655, -0.030937839299440384], [-1.4534504413604736, 1.5877515077590942, 0.28032374382019043, 0.46985962986946106, -0.40859249234199524, 0.5461751222610474, -0.5864737033843994, 1.4033607244491577, 1.3743858337402344, -0.5971161723136902, -0.7790312767028809, -1.4193953275680542, -1.368478536605835, 1.3592506647109985, 0.8901622295379639, 1.6870664358139038, 0.6417639851570129, 1.0144795179367065, 0.8079535961151123, -0.34841135144233704, 0.34505653381347656, 1.0906347036361694, 0.29581889510154724, -0.808639407157898, -0.8305936455726624, 3.1173787117004395, -0.7110452055931091, -0.5067533254623413, 0.22049134969711304, 0.07538223266601562, 1.1806684732437134, -0.2449997514486313, 0.8252784609794617, 0.2478552609682083, 0.25034207105636597, -0.01222191657871008, -0.7047547698020935, -0.348072350025177, -0.9887223243713379, -0.27914971113204956, 0.2795493006706238, 0.38812455534935, -0.9812436103820801, -0.5982367992401123, -0.18218640983104706, 1.467629313468933, 0.3585329055786133, 0.5158430337905884, 1.3789113759994507, 0.6407270431518555, -0.3610503077507019, 1.8214436769485474, 0.02097543701529503, 1.3352537155151367, 0.5559730529785156, 0.7857832312583923, 1.4477472305297852, -0.2973117232322693, -1.2787977457046509, 0.4580390751361847, 0.08680824935436249, -0.12419930845499039, 0.14738775789737701, -1.4336656332015991], [0.11309107393026352, -0.1667010337114334, 0.4109669029712677, 0.01962805911898613, -0.252989798784256, 0.7525773644447327, 1.1807031631469727, 0.009176774881780148, -0.7648316025733948, -0.5947780013084412, 1.282982349395752, -1.5955275297164917, 0.09632227569818497, 0.3107394874095917, 0.49746307730674744, -1.9795383214950562, 0.22296489775180817, -2.180830240249634, -0.5805161595344543, -1.0273971557617188, -2.083286762237549, 1.620255947113037, -0.480985552072525, -1.7515766620635986, -1.2753428220748901, -0.31034770607948303, 0.5307537913322449, -0.14811673760414124, -1.3603079319000244, 1.409977912902832, 0.2054949551820755, 1.5031423568725586, 1.2679322957992554, 0.21525603532791138, 1.2690597772598267, -0.1754191517829895, 0.9741461277008057, -1.6886873245239258, -1.8811297416687012, -1.1591875553131104, 0.45454639196395874, 0.07814662903547287, -0.7875313758850098, -0.6560848355293274, -0.6241730451583862, 0.0701427310705185, 1.0197868347167969, 0.9252178072929382, -0.246258944272995, -0.0731838271021843, 0.5370272994041443, -1.1194283962249756, -0.037257060408592224, 0.5378785133361816, 0.4163036048412323, -0.48437991738319397, -0.8481769561767578, -0.9332056641578674, 1.3815529346466064, 0.004185743164271116, -1.2492139339447021, -2.0324342250823975, -0.0012230048887431622, 1.484907627105713], [-0.2852284014225006, -0.8980448246002197, 0.7887653112411499, -0.333074152469635, -0.15907470881938934, 0.2706158459186554, -0.3367537260055542, 0.6610546708106995, 0.803743839263916, -0.10491452366113663, -0.5684873461723328, 0.2959232032299042, -2.5477612018585205, -0.43735185265541077, -0.574709951877594, 0.7080899477005005, -1.0851657390594482, 1.0531055927276611, -0.1559278517961502, 0.3917694091796875, 1.2776176929473877, 0.13758280873298645, -0.6245824694633484, 0.2689867615699768, 0.2914106845855713, 1.9139188528060913, -0.0689375251531601, 0.9147412180900574, 1.3252241611480713, 1.1022758483886719, -0.22185564041137695, -1.1573714017868042, 1.0608969926834106, 1.7141567468643188, 0.6813824772834778, 1.2061262130737305, 0.19358403980731964, 0.890209436416626, 1.3827964067459106, 0.07748039066791534, 1.2969330549240112, 0.04575445130467415, -0.830608069896698, 0.469716876745224, -0.07496832311153412, 0.5358691215515137, 0.6505110859870911, -0.08989182859659195, 0.4967024028301239, -0.3564531207084656, 0.8994153141975403, -0.8239049911499023, -1.280012845993042, 1.1010522842407227, -1.8915107250213623, -0.19339871406555176, -0.1414763629436493, 0.8939318060874939, -0.1832791119813919, -0.1838146150112152, -1.7774192094802856, 0.9990590214729309, -1.022274374961853, -0.005682750139385462], [-0.777435302734375, 1.1304495334625244, -0.9250909686088562, -1.3490660190582275, 1.298081874847412, -0.6556217670440674, 0.22845375537872314, 0.4188864231109619, -1.5086030960083008, 1.0030704736709595, -1.0337074995040894, 0.061122260987758636, 0.4351845383644104, -0.08686766773462296, -0.8916633725166321, 0.929705023765564, -0.8886168599128723, -1.322123646736145, 1.171439528465271, 1.230987787246704, -1.878366470336914, -0.22098511457443237, -0.012426040135324001, -2.8408007621765137, -0.5758945941925049, 1.137115240097046, -1.0194538831710815, 1.3009381294250488, 1.973767876625061, -0.1630055457353592, -0.743348240852356, 0.05804387107491493, -1.4529953002929688, 1.2814292907714844, 0.4957069754600525, -1.427046298980713, -0.8022620677947998, -0.3241294324398041, 1.0888526439666748, 0.836169958114624, -0.6527667045593262, 0.17185263335704803, -0.44802284240722656, 0.4788268506526947, -0.27778080105781555, 2.1177985668182373, 0.46831974387168884, -0.26938575506210327, -1.0667262077331543, -1.4131758213043213, 0.7295089960098267, 0.8277548551559448, 1.5849480628967285, -1.0354958772659302, 0.7566903829574585, -1.2410223484039307, 0.6890734434127808, 0.573789119720459, 0.23780126869678497, -0.3094487488269806, -0.486259788274765, -0.37309759855270386, -1.9478912353515625, 0.67911696434021], [-1.1434104442596436, 0.15981535613536835, -0.05379559099674225, 0.725807249546051, 0.07691055536270142, 0.02950834482908249, 0.30519717931747437, 1.0108662843704224, -0.3782815635204315, -0.8681812882423401, 0.0511961355805397, -0.08230289816856384, 0.7649745941162109, 0.4328940808773041, 0.19973082840442657, 1.1731125116348267, 0.34558340907096863, 0.675385057926178, -0.3522491157054901, 1.5677682161331177, 0.7143169641494751, -0.8778802156448364, -0.8327956199645996, 1.8438302278518677, 0.2975374460220337, 1.8098818063735962, 2.439117670059204, 0.2193886637687683, 0.38346973061561584, 0.8977211713790894, -0.8163151144981384, -1.7417490482330322, 1.9581966400146484, -0.7247838973999023, -0.04142562672495842, 1.5978004932403564, 0.17470817267894745, 1.0384231805801392, 0.4251341223716736, -1.183004379272461, -0.6662361025810242, 0.19472649693489075, -0.8675304055213928, 1.0816898345947266, 0.4670279026031494, -1.485260248184204, 0.5791328549385071, -0.4230351448059082, -0.22250095009803772, 0.8114640712738037, -1.786007285118103, -0.8571941256523132, 0.35326680541038513, -1.4082790613174438, 1.3620105981826782, -3.2235703468322754, 1.1367980241775513, 0.795490026473999, -0.6096446514129639, 0.11581864207983017, -0.70713210105896, 1.930295705795288, 0.01676533743739128, 0.5297750234603882]]], [[[0.3240806460380554, -1.3678534030914307, -1.9216625690460205, 0.19571027159690857, 1.455841302871704, 0.6762587428092957, -0.1477973461151123, 0.4661216139793396, -0.3326398432254791, 1.3233586549758911, 0.6402652859687805, 0.3462657034397125, 2.1564290523529053, -0.5545221567153931, -0.9038958549499512, -0.21247628331184387, -0.5039451718330383, -0.5528506636619568, 0.6496098041534424, 0.562119722366333, -1.7146941423416138, -0.6664971113204956, -0.5114247798919678, -1.850893259048462, 0.6288632154464722, 0.12625189125537872, 0.480479896068573, -0.6603986024856567, -0.6047845482826233, 0.23425008356571198, 0.062325816601514816, -0.47108694911003113, 0.4740225076675415, -0.7133165001869202, 0.9183318614959717, 0.39484554529190063, -0.6064939498901367, -1.3082178831100464, 0.27151912450790405, 1.0833890438079834, 0.01165634673088789, 1.4111902713775635, 0.7336990833282471, 1.2809360027313232, -1.5086115598678589, -0.6716793775558472, 0.09154467284679413, 0.22495202720165253, -0.1563691645860672, -0.4259496331214905, -2.1936912536621094, 1.1164088249206543, 0.751190721988678, -0.49839040637016296, 0.022532103583216667, -0.6546835899353027, 0.9935736656188965, -0.25065097212791443, 1.1508915424346924, -0.5228586196899414, -0.9375754594802856, -1.6657934188842773, 0.416054904460907, -0.648723304271698], [0.5414428114891052, 1.0652812719345093, 0.307271271944046, 1.4959256649017334, -0.4146920144557953, -0.8145109415054321, 0.27764415740966797, -0.16281543672084808, 1.5854729413986206, -1.7585898637771606, -1.1065566539764404, 1.2899729013442993, -0.803180992603302, -0.24482229351997375, -0.5156071186065674, -0.06362397968769073, 2.0720388889312744, 1.0703520774841309, -0.007531690411269665, 1.9056308269500732, -0.9686286449432373, 0.176248237490654, 1.9178740978240967, -1.2776687145233154, -0.9462248682975769, -0.955971896648407, -0.37081918120384216, 0.7791385054588318, 1.5112498998641968, 1.1446962356567383, 1.322918176651001, 0.3622460663318634, -0.541554868221283, -0.9122899174690247, 0.002183526288717985, -0.913764476776123, -1.2283377647399902, 0.542465329170227, -0.8754155039787292, -0.3201415538787842, 2.069988489151001, 0.48145902156829834, 0.5056977868080139, -0.2505156695842743, -0.9197723865509033, 0.5628900527954102, -0.4241677224636078, -0.18308478593826294, -0.6374444961547852, 0.3763900101184845, 0.22274883091449738, -2.2196829319000244, -0.9814381003379822, -0.2099340707063675, 1.8388869762420654, -0.9947884678840637, 0.3375232219696045, 1.0879923105239868, 1.0771023035049438, -0.5437743067741394, 0.8907426595687866, 0.23739676177501678, 0.05376880243420601, -0.5204508304595947], [-1.803023099899292, -0.5493133664131165, -1.5938389301300049, 0.8351545333862305, 0.38948968052864075, 1.7228944301605225, -1.534836769104004, -1.6342179775238037, -0.2599266767501831, 0.13811154663562775, 0.8086139559745789, -1.226036548614502, -1.8970955610275269, -1.085283875465393, 3.300605297088623, -1.1031277179718018, -0.7220099568367004, 0.7452561259269714, 0.49810877442359924, -0.5398253798484802, -3.1994268894195557, 0.5183766484260559, -0.6529751420021057, 1.2164244651794434, -0.03439180552959442, 0.47351735830307007, -2.005758285522461, 0.007817280478775501, -0.15269735455513, -0.5296470522880554, 0.3557344377040863, 0.5215749740600586, -0.49737805128097534, 0.5819966793060303, 1.4654425382614136, 0.4993889629840851, -0.1841312050819397, 0.27192771434783936, -0.22531485557556152, 0.25572875142097473, 0.7854230403900146, -0.35654959082603455, 0.6618443727493286, 1.660173773765564, -0.22056207060813904, -0.7647355198860168, -0.4060629606246948, 0.6630030870437622, 0.3919092118740082, -0.5968066453933716, -2.6187360286712646, 0.4890161454677582, 0.43832284212112427, 0.41469696164131165, -0.7040902972221375, -2.2411065101623535, -0.38688182830810547, -0.772457480430603, 0.5731558799743652, 1.5451154708862305, 0.8699582815170288, -0.28885456919670105, 0.8150743246078491, -0.5147525668144226], [0.774875819683075, 0.10919222980737686, -0.4634404480457306, 0.4234868586063385, 0.19661743938922882, 1.438188910484314, -0.019295869395136833, -0.08849328756332397, 1.4673798084259033, 0.43405938148498535, 0.30220746994018555, 2.1054508686065674, -2.3599815368652344, -1.3833439350128174, -0.4919191598892212, 1.4510550498962402, 1.6868363618850708, 0.48839855194091797, -0.43337148427963257, -1.1848162412643433, 0.7935181856155396, 0.6275580525398254, -0.32427501678466797, 1.303299903869629, 1.9426275491714478, -0.6267013549804688, 0.5900862812995911, -1.0156712532043457, 0.5118036270141602, -1.0017659664154053, 0.3267207443714142, 2.1813621520996094, -0.3241310119628906, 1.7359906435012817, 1.6164398193359375, 1.4702837467193604, -0.14000746607780457, -1.18061363697052, 0.2884925603866577, 0.87519770860672, -0.33240559697151184, -0.6647538542747498, -1.0363606214523315, 3.231355905532837, -1.8582448959350586, 0.9991834759712219, -0.3084733784198761, 1.367713451385498, 1.4582114219665527, -0.8536527156829834, 0.1040748804807663, -0.15155692398548126, 0.736409068107605, 0.04138306900858879, 0.9767021536827087, 0.668829083442688, -0.4192309081554413, 1.2396997213363647, 0.6507455110549927, 0.6967126131057739, -0.25227710604667664, 0.8942233920097351, -0.23650552332401276, 1.4801119565963745], [0.5928309559822083, -2.1389002799987793, 0.7977235317230225, 0.7035781741142273, 0.2202211618423462, 0.987251877784729, -1.325478434562683, 0.5145424008369446, -0.5983438491821289, 0.378528356552124, -0.5617437362670898, -1.1301753520965576, -0.1791311353445053, -1.5344429016113281, 0.18851712346076965, 0.8742617964744568, 1.4413152933120728, 0.5508870482444763, -0.32710781693458557, -0.0483885183930397, 0.02827717736363411, 1.3887914419174194, -0.8073453903198242, -0.38403618335723877, -1.9343458414077759, 1.3846020698547363, -1.0681082010269165, -1.1452676057815552, -1.821010708808899, 0.3776572048664093, -0.15825296938419342, -0.2503286898136139, -1.2847799062728882, -0.44067302346229553, 1.2569974660873413, -1.4418058395385742, 0.20821894705295563, 1.4275099039077759, -1.0566538572311401, 0.7179573774337769, 0.7354019284248352, -0.7170832753181458, -0.019570140168070793, 0.728198230266571, 0.9398220181465149, -0.5282438397407532, -1.4046063423156738, 0.2257322371006012, 0.22266478836536407, 0.7346832752227783, 1.4495301246643066, 1.185120940208435, 0.9539763927459717, -0.22919951379299164, 1.3462656736373901, 1.2469595670700073, -0.2644408643245697, -0.39962276816368103, -1.2755473852157593, -0.3912849724292755, 0.10604032129049301, -0.13747794926166534, 0.692025899887085, 0.14151883125305176], [-0.7692358493804932, 1.3322162628173828, -1.5929110050201416, 1.1265217065811157, 0.16440390050411224, 0.37914565205574036, -0.561843752861023, 0.23158057034015656, 1.716468334197998, -0.5230008959770203, 0.4197006821632385, 0.23364906013011932, -2.9572997093200684, -0.47725361585617065, 0.40757590532302856, -0.5801024436950684, -0.09796924144029617, -1.1183990240097046, 1.7162610292434692, 0.39009061455726624, -0.10636631399393082, -0.6637086868286133, 1.0307412147521973, 0.22040659189224243, 0.8949521780014038, 0.6844693422317505, -1.272091269493103, 0.5709240436553955, -2.308544158935547, -2.6406824588775635, -0.23886987566947937, 1.748420000076294, 0.5827145576477051, 0.5868804454803467, -0.831283450126648, -0.13493883609771729, 0.5361659526824951, -2.089913845062256, -0.7653740048408508, -0.4763548672199249, 0.9014745354652405, 1.229602575302124, -0.5839805006980896, 1.5176352262496948, 0.16222669184207916, -0.06325722485780716, 1.3406933546066284, -0.7634558081626892, -0.6397113800048828, -1.2340511083602905, -0.7788592576980591, -2.221921920776367, 2.488284111022949, 0.7014798521995544, 1.9129794836044312, -1.5365393161773682, -1.831802248954773, 1.4670580625534058, -0.2779097259044647, -0.34215447306632996, -0.48074328899383545, -1.3485223054885864, -0.734042227268219, 2.119088888168335], [-1.0937628746032715, 1.46792733669281, 0.38787829875946045, 0.8205649852752686, 1.162490725517273, -0.9827112555503845, 0.8757071495056152, 0.4536117911338806, 2.746666431427002, 0.4512075185775757, 1.0482177734375, 0.2391623854637146, 0.40618932247161865, 0.31035521626472473, 0.009234639815986156, 1.0952003002166748, 0.35260236263275146, 0.5628849267959595, 0.937782883644104, -0.6196615695953369, -0.24927493929862976, 0.357412725687027, 0.5408771634101868, 1.8080708980560303, -2.0749099254608154, -0.749607503414154, -0.9511321187019348, 0.0850684717297554, -0.23895052075386047, -1.709924340248108, -0.9410361647605896, -0.9363610148429871, -1.7693899869918823, -1.0548179149627686, -0.6413883566856384, 1.0338715314865112, 0.08378145843744278, -0.5365338921546936, -0.7737843990325928, -0.24408479034900665, 1.1843680143356323, 1.2808606624603271, -0.8704200983047485, -0.37676674127578735, 0.1332108974456787, 0.2899605929851532, -1.0702589750289917, 0.5765387415885925, 0.8096770644187927, 0.8899059295654297, 1.8787288665771484, 0.9865936040878296, -0.1559108942747116, -1.419926643371582, -0.06226211413741112, -0.6617076992988586, 1.104094386100769, -0.08852441608905792, -1.2490501403808594, -0.5025092363357544, 1.4913378953933716, -0.6656333208084106, 0.1882135272026062, 1.0533355474472046], [-0.30486148595809937, -1.4221748113632202, -0.8767101168632507, -0.6313964128494263, 0.2837396562099457, 1.0913490056991577, 1.4976435899734497, -1.872179627418518, -0.0200730599462986, 0.6831691265106201, -1.4714542627334595, 1.5139306783676147, -1.6283832788467407, -0.523383378982544, -0.4690706431865692, 0.9417255520820618, -0.4545987844467163, -0.32279106974601746, 0.560083270072937, -0.40471139550209045, 0.8774431347846985, -0.8093648552894592, 1.105682611465454, 0.05309370160102844, -0.3286221921443939, 0.4068012535572052, 0.3482084274291992, 0.188704714179039, 0.7564824819564819, -0.13857129216194153, -2.298637866973877, 0.5332653522491455, 0.52315354347229, -1.4307637214660645, -1.1740083694458008, -0.022053195163607597, 0.0750250518321991, 0.7971007823944092, 1.8140162229537964, 2.144070625305176, -0.4092775583267212, -0.30946987867355347, -0.3459314703941345, 0.27465569972991943, 2.0760715007781982, -0.07451585680246353, -0.745926022529602, -0.28910887241363525, -0.18030573427677155, -0.5830947756767273, -1.0252373218536377, -0.8923237919807434, 0.8154400587081909, -1.573602557182312, 1.4361863136291504, 2.1328213214874268, -0.7587401866912842, -0.32067596912384033, -0.08545047789812088, 1.3760613203048706, 0.7047609090805054, 1.0328459739685059, -1.2356033325195312, -0.743843138217926], [1.717306137084961, -1.3495492935180664, 0.15143030881881714, -0.8831983208656311, 0.834039032459259, -0.5362573862075806, -2.2259268760681152, -1.1682049036026, -0.3086549937725067, -0.33190372586250305, 1.5183475017547607, 0.41749194264411926, -1.051293969154358, 0.7881765365600586, -0.14137689769268036, 2.0652332305908203, -0.28867900371551514, -0.6490789651870728, 0.29939067363739014, 1.7889533042907715, -0.39325469732284546, 1.5432764291763306, 0.07910846173763275, 0.2537718117237091, 1.3100544214248657, 1.119065523147583, 0.6159331202507019, 1.9124252796173096, -0.4499320685863495, -2.454457998275757, 0.7417799830436707, 0.231427863240242, -0.19995218515396118, 1.225999355316162, -2.2055985927581787, -0.30375638604164124, 1.310563325881958, 0.3109147846698761, 1.2712477445602417, 0.2703321874141693, -1.2198766469955444, -1.2181570529937744, 0.1788962334394455, -1.3295496702194214, 1.6056102514266968, -0.8961048722267151, -1.82364022731781, -0.2922292947769165, 0.504667341709137, -0.11795013397932053, -0.22850355505943298, -0.49848753213882446, -1.829942226409912, 0.11415518075227737, 0.767385721206665, -1.0313938856124878, 0.4766293168067932, -1.7563821077346802, 0.16852223873138428, -1.0866023302078247, 0.8074703216552734, -1.4986063241958618, -1.2568796873092651, -0.37685462832450867], [-0.08961267024278641, -0.35765644907951355, 0.6653513312339783, -0.05698070302605629, -0.35984256863594055, 1.6284034252166748, 0.9158811569213867, 0.20308683812618256, 0.327973872423172, -1.8808190822601318, -1.306904673576355, 2.120358943939209, -1.5506370067596436, 0.9441961646080017, 0.39116573333740234, -0.437399685382843, 2.5407304763793945, -0.2983216941356659, 1.4734892845153809, -0.6977433562278748, 0.2806962728500366, 0.3104807436466217, 0.19660913944244385, 1.1373932361602783, 0.269223153591156, -1.3233532905578613, 0.17706887423992157, -1.7846089601516724, 1.3841500282287598, 0.964046835899353, 0.8791561126708984, -0.24478214979171753, 0.4822591245174408, -2.262791395187378, -0.6630811095237732, 0.11351140588521957, -2.3652944564819336, 0.13565871119499207, -0.4509366750717163, 0.26246944069862366, -0.633801281452179, 1.3326425552368164, -0.10827410966157913, 1.1771965026855469, 0.8529519438743591, -1.2928738594055176, -2.1866183280944824, -0.38439106941223145, 0.27305397391319275, 0.31291475892066956, -0.519102156162262, 2.1992383003234863, -0.1357472687959671, 1.495552659034729, -1.75140380859375, 0.4746882915496826, -0.16530963778495789, 1.0645830631256104, 2.5297794342041016, -0.6903075575828552, -0.5940001010894775, -0.12490882724523544, -0.31560778617858887, -0.024828974157571793], [0.040798645466566086, -0.587444007396698, 0.007899089716374874, -0.23677779734134674, 1.3452060222625732, -0.6073068380355835, 0.8240179419517517, 0.5986742973327637, 0.16685166954994202, 1.1304454803466797, -1.9383450746536255, 0.4103485345840454, -0.7540855407714844, -2.3641984462738037, -0.39639726281166077, 0.3113541603088379, -2.0502066612243652, 0.26978227496147156, -1.0393439531326294, -0.8783811330795288, 1.5180894136428833, 1.1839470863342285, 0.6623528599739075, 0.39367109537124634, -0.9402511119842529, 1.8601704835891724, 1.109928011894226, 0.7128311395645142, 1.45442533493042, 0.21132813394069672, 0.05869413912296295, 0.7973194122314453, 0.8306521773338318, 1.6861491203308105, -0.38253119587898254, 1.8551812171936035, 0.10898423939943314, 1.0866895914077759, 0.37622225284576416, -1.8155654668807983, -0.13791286945343018, -0.14933717250823975, -0.3754732310771942, 2.5752830505371094, -0.3785961866378784, 0.5970938205718994, 0.7128015756607056, -0.4201318323612213, -1.430572748184204, -0.9412431716918945, 1.5892817974090576, -0.7443103790283203, 0.3464356064796448, 0.3096601366996765, -1.1070961952209473, 0.832427442073822, 0.5256113409996033, -1.3402981758117676, -0.9707240462303162, -0.4504861533641815, 1.0859100818634033, -0.7349144816398621, -2.0300445556640625, 1.7827898263931274], [-1.0944197177886963, -0.6299670934677124, -0.6534979343414307, -2.067902088165283, 1.3160443305969238, 0.9990230202674866, 1.0820132493972778, 0.2403668910264969, 3.2732276916503906, -1.4326262474060059, 0.08576612174510956, -0.32789289951324463, -2.4104762077331543, -0.8524674773216248, -0.26041722297668457, -0.1670616716146469, -0.3967569172382355, -0.07948307693004608, 0.16059094667434692, -0.2841622829437256, 1.280636191368103, -0.08225618302822113, 1.3942691087722778, 1.501094102859497, 0.18753927946090698, -0.8707332611083984, 0.4392290711402893, 1.3750724792480469, -2.310857057571411, -0.30985191464424133, 2.475829601287842, -0.2092994749546051, -0.7502995729446411, -0.7794491052627563, -0.622065007686615, -0.5144359469413757, -1.3651959896087646, 0.13037057220935822, 0.013852369040250778, -1.431694507598877, 1.274221658706665, 1.11076021194458, -0.5641658902168274, -0.28695979714393616, 0.4127286672592163, 0.015705671161413193, 0.5900428891181946, 1.1439707279205322, 1.636017918586731, 1.0924015045166016, -0.1641114503145218, -0.32468101382255554, -1.5454367399215698, 0.15364602208137512, 1.716671347618103, 0.6499969363212585, -0.6306609511375427, 0.37544941902160645, -0.39376747608184814, 0.10394161194562912, -0.6198724508285522, 1.7189702987670898, -2.2070703506469727, -1.0571423768997192], [0.652655839920044, -2.2411932945251465, -0.4433800280094147, -1.314754605293274, 0.3448668420314789, 0.009701893664896488, -0.09811045974493027, -0.00970652885735035, -0.23702266812324524, 1.98464035987854, -0.35759878158569336, -1.0788922309875488, 0.061641838401556015, -1.590362310409546, -1.123458981513977, 1.0264973640441895, 0.17820684611797333, -0.11718665063381195, 1.1401007175445557, -0.9621736407279968, -1.9316285848617554, 1.063501000404358, -0.27034300565719604, 2.513941764831543, -1.6334583759307861, 0.0718780979514122, -0.5751168131828308, 0.2411363124847412, -0.08716582506895065, -0.36485517024993896, 1.592492938041687, 1.1292383670806885, 0.40056121349334717, 0.6978684663772583, 0.36867234110832214, -1.8088046312332153, -0.8574612736701965, 0.36830082535743713, 0.21920114755630493, 1.808619499206543, -0.7493268251419067, -0.130567267537117, 0.06377211958169937, 0.2709585130214691, 1.3073303699493408, -0.8643958568572998, 1.0922484397888184, 0.43319085240364075, -1.488219976425171, -1.3377563953399658, -0.08540556579828262, 1.8580971956253052, 0.40956032276153564, 0.6514831781387329, 0.3587650954723358, -1.221747636795044, -0.6507357358932495, -1.6597981452941895, 0.8998413681983948, -0.42286476492881775, 1.3308764696121216, -0.07909727096557617, -1.501686453819275, -0.4217556416988373], [1.0586249828338623, -0.3201506435871124, 0.08488515764474869, -1.5912894010543823, -0.38501161336898804, 0.7834437489509583, -0.09614534676074982, 0.23204046487808228, 0.29744574427604675, -0.8104091286659241, 0.07524097710847855, -0.07016508281230927, -0.7050802707672119, 1.842074990272522, 1.6934869289398193, 1.407400369644165, -0.04928077384829521, -1.5695983171463013, 0.8214635252952576, -0.670816957950592, 0.8401554226875305, -0.374175101518631, -0.6954014897346497, -1.9354413747787476, 0.34189870953559875, 0.6357929706573486, 0.6959586143493652, 0.9965394139289856, -1.6866213083267212, 0.19383804500102997, 0.9462235569953918, 1.1370705366134644, -1.6307276487350464, 0.32061833143234253, 1.1263102293014526, 0.7430835962295532, -0.8926247358322144, 1.0083364248275757, 0.331466943025589, 1.539768099784851, -0.6651175022125244, 0.20702241361141205, 0.06969541311264038, 0.7518766522407532, -1.419697880744934, -0.04124514013528824, -1.3127243518829346, -0.1893422156572342, -0.562842607498169, -0.029891226440668106, 0.8500314950942993, 0.40397512912750244, 0.5295793414115906, 0.4711354374885559, -0.10630005598068237, -0.6290395259857178, 0.359324187040329, -1.782119870185852, 0.2458002120256424, -0.5170716047286987, -0.7352774143218994, 0.08773908764123917, 0.10938441008329391, -1.110041618347168], [-0.013981220312416553, 0.5594531297683716, -0.7203802466392517, 0.7293131947517395, -1.4648712873458862, -0.8783168792724609, 0.7245556712150574, -0.2732691168785095, -0.5666722655296326, 0.508558988571167, 0.7557640075683594, -0.053451597690582275, 0.37059924006462097, 0.06824751198291779, -0.5704014897346497, 0.26177820563316345, -0.2726665139198303, -2.108954906463623, -0.5292848944664001, 0.5081583261489868, 0.03297705203294754, -0.038254667073488235, -0.6426332592964172, -1.5597003698349, -1.0235408544540405, -1.3948842287063599, -1.5805702209472656, 0.449943870306015, -0.5095123052597046, -1.2747927904129028, 0.4535273611545563, 0.7786860466003418, -1.5304663181304932, 0.6446101665496826, -1.300795555114746, 2.562025308609009, 0.31455302238464355, -2.0598866939544678, 1.1535439491271973, -1.0212448835372925, 0.4341508448123932, 1.725103497505188, 0.9578354358673096, -1.6462304592132568, 0.11101148277521133, -0.6522006392478943, 0.2298366129398346, -0.34214988350868225, -0.14784203469753265, 1.5492324829101562, -0.644245982170105, 1.2221477031707764, -0.9852613806724548, -0.23082248866558075, -1.5296506881713867, 1.1576136350631714, 0.07138579338788986, -1.3866899013519287, 0.10306810587644577, -0.41614899039268494, -0.47738492488861084, -0.5634847283363342, 0.6296229958534241, -0.6759562492370605], [1.0127488374710083, 0.6117656230926514, 0.9512954354286194, 1.10902738571167, 0.4446694552898407, -0.7094317674636841, 0.8943852782249451, -1.4269027709960938, -0.6764688491821289, 1.1463491916656494, -1.8082952499389648, -0.8859174847602844, -1.3372513055801392, 0.3736979067325592, -1.02860689163208, 0.19449077546596527, 0.043949492275714874, 1.979201078414917, 2.0958123207092285, -0.8327865600585938, -0.6717298030853271, -0.05900539085268974, -0.3766714334487915, -0.3415672481060028, -0.15892665088176727, 0.9098086953163147, 1.372442603111267, -1.4148451089859009, 0.5944443941116333, 0.14774231612682343, -0.9890608787536621, -0.05647042393684387, -0.5188241600990295, 0.10929341614246368, 0.10399053990840912, -1.7016546726226807, -0.5163238644599915, 0.23603498935699463, -0.6285950541496277, -3.1091034412384033, 1.3137089014053345, -0.1568969339132309, 0.6279906034469604, -0.6560253500938416, -0.8922040462493896, 0.9882558584213257, 0.10492151975631714, -0.486226350069046, 0.3620564341545105, -0.3193877935409546, 0.2488717883825302, 0.8613300323486328, 0.04190696403384209, -0.4379061758518219, 0.6884281635284424, 1.204564094543457, 0.25200197100639343, 0.03607378154993057, 0.04703017324209213, -0.6341204643249512, 0.6808624267578125, -0.39703401923179626, -0.3545634150505066, 0.4031873345375061]], [[1.4778506755828857, 1.1717076301574707, -1.062908411026001, 1.439521312713623, 0.8025926947593689, 0.1514747589826584, 0.9817519187927246, 0.2966782748699188, 1.922666072845459, -0.7173696756362915, 1.4971003532409668, -0.41782450675964355, -0.2691618502140045, 0.8883659839630127, -1.9553759098052979, 0.9094483256340027, 1.319114327430725, -1.680073618888855, -0.633281946182251, 1.060203194618225, -0.9616311192512512, -1.3203486204147339, 1.8076236248016357, -0.3892544209957123, 1.6459420919418335, 1.0479435920715332, 0.391729474067688, -0.7843952178955078, 1.1159218549728394, 0.5206010341644287, -0.5596349239349365, -0.7297086119651794, -0.33412590622901917, -1.1359909772872925, -0.4860326647758484, 0.4842219650745392, -0.8894041180610657, -0.35507723689079285, -0.30706682801246643, -0.13724498450756073, -0.6799649000167847, 0.4193349778652191, 1.1986234188079834, -1.2843053340911865, -0.5419999361038208, -0.1517971009016037, 1.1768007278442383, -0.13478809595108032, -0.522971510887146, 0.8836696743965149, -0.1663135439157486, -1.0703206062316895, 0.25000908970832825, 0.10676278918981552, -0.6321485638618469, -1.6834800243377686, -0.5440505146980286, 0.3968663215637207, 0.3381110429763794, -1.5034687519073486, -1.1678626537322998, 0.9026063680648804, -1.8426244258880615, 0.8732308149337769], [0.6527636647224426, 2.015495538711548, -1.6676045656204224, 0.8358821272850037, -0.6080018281936646, 0.027867209166288376, -1.0483587980270386, 0.9192641973495483, 0.5542119145393372, -1.1629598140716553, -1.4636321067810059, 0.3239675760269165, -0.47687599062919617, 0.19949114322662354, 0.18514060974121094, 0.35669150948524475, 0.0713285431265831, -0.5130950212478638, 1.0061553716659546, 0.7355831265449524, -0.035393279045820236, 0.38582512736320496, -0.008950408548116684, -0.10535839200019836, -0.4893485903739929, 1.3763456344604492, -0.5269622206687927, 1.3603503704071045, 0.6695204973220825, 0.5828642249107361, 0.7016631364822388, 0.44255688786506653, -1.2608070373535156, -1.492821455001831, -0.7636751532554626, -0.5978853702545166, -0.04890973120927811, -0.17302407324314117, 0.3164033889770508, -1.7196252346038818, 0.09780445694923401, -0.7836015820503235, -0.7513781189918518, -0.7647578120231628, -0.08814771473407745, -1.0017905235290527, 0.6183966994285583, 0.6447989344596863, -0.7979119420051575, -0.7958347797393799, -0.11332419514656067, 1.109632134437561, 0.5300443768501282, -0.7856763601303101, -0.2061791568994522, -1.3263742923736572, -0.5754131078720093, -1.1959589719772339, 0.4208774268627167, -0.36765044927597046, -0.17918722331523895, -1.7079375982284546, -0.5871677398681641, -0.522705614566803], [0.9729191660881042, -1.8086309432983398, -1.0716532468795776, -1.0886861085891724, 0.29471346735954285, -0.7221387624740601, 1.069154143333435, 1.445449948310852, -2.0357542037963867, 1.177430272102356, 0.6065773367881775, 0.7102747559547424, 0.44544118642807007, -0.3412761390209198, -0.5641161799430847, -0.6523860692977905, -0.9445432424545288, 0.22162765264511108, -0.13771748542785645, 0.12488869577646255, 0.09333427250385284, -1.726348638534546, -0.1268308460712433, -0.8091046214103699, -0.9525368213653564, -0.1212654709815979, 0.7907422780990601, -0.2882816791534424, 0.22042728960514069, 1.5082272291183472, 0.6470072865486145, 0.5331809520721436, -0.7266383171081543, 0.8471091985702515, -0.34290778636932373, 1.5095937252044678, 1.9556955099105835, -0.6744992733001709, 0.3035876750946045, 0.2052280753850937, 1.5523861646652222, -1.2408734560012817, 1.2250313758850098, 1.1440564393997192, 0.16911372542381287, 0.6299594640731812, -0.015245816670358181, 0.40197306871414185, -0.0847088173031807, -1.29597008228302, -0.47954854369163513, -0.4434421956539154, 0.6289181709289551, -0.3469647467136383, 0.5434751510620117, 1.1056432723999023, -0.30568599700927734, 0.5861474871635437, -1.2443137168884277, 0.20009282231330872, 1.0718218088150024, 1.014975905418396, 3.5673112869262695, -0.4409671127796173], [0.18436837196350098, 0.13930298388004303, -0.5929490923881531, 0.028835270553827286, -1.1736884117126465, -0.2304847538471222, 3.0203986167907715, -0.6822523474693298, 0.9831365346908569, 1.3868460655212402, 1.5015201568603516, 0.7088115811347961, 0.3626127243041992, 0.9326211810112, 1.060015082359314, -1.435847282409668, 1.5338211059570312, 0.1744007170200348, 1.0180249214172363, 1.0685348510742188, 0.4230239689350128, -1.1126775741577148, -1.6690762042999268, 0.9320728778839111, -0.6101217269897461, -1.4374793767929077, -0.4998711347579956, -0.002834590617567301, 1.563658356666565, -1.0722678899765015, 0.46185317635536194, 0.15708063542842865, -0.4446231722831726, -1.014357566833496, 1.4231094121932983, -0.2042001336812973, 0.2998610734939575, -0.2489088475704193, 1.818052053451538, 0.31657183170318604, -0.4377683401107788, 0.9046235084533691, 0.15963315963745117, -0.9867634177207947, -1.4866127967834473, -0.6034660935401917, -0.550509512424469, -1.5842849016189575, -2.1086671352386475, 0.08413279801607132, 0.003395203733816743, 0.5348941683769226, -1.2621228694915771, -0.09992027282714844, -0.0009984277421608567, -0.4402512311935425, -0.9596166610717773, -0.07679780572652817, -0.6163137555122375, -0.7154316902160645, -0.4595522880554199, -1.0566318035125732, -1.1085840463638306, -0.2633673846721649], [-0.10762066394090652, 0.6040247082710266, -0.3608371615409851, -0.9232916831970215, -1.2750123739242554, 0.7855157852172852, -0.6493817567825317, -1.1817185878753662, 1.3302538394927979, 0.920102596282959, -0.6697805523872375, -0.19037850201129913, -0.9396185874938965, -0.22762344777584076, 1.2986669540405273, -0.2039039582014084, 0.9434242248535156, 1.2223377227783203, -1.909683346748352, -0.7994086742401123, 1.2624633312225342, -0.6573380827903748, -0.6673133969306946, -0.7998536825180054, 0.8196184635162354, -0.17181019484996796, 2.3786094188690186, 0.2351534515619278, -2.117511034011841, -1.295159935951233, 0.09008654206991196, 1.8315507173538208, 0.9193454384803772, -0.7152693271636963, -0.6145361065864563, -1.6496227979660034, 0.9659966230392456, -1.6817933320999146, 0.7210935354232788, 0.8792908787727356, 0.3623609244823456, 1.1138193607330322, 2.1475753784179688, -0.8094921708106995, 1.7383432388305664, -2.136404275894165, -0.4001534879207611, -0.21634535491466522, -0.1001284196972847, 1.1321702003479004, -0.23220698535442352, -0.9391445517539978, -0.9851633310317993, -0.4003034830093384, 0.5093330144882202, -1.1112287044525146, -0.7496314644813538, -0.6930408477783203, 0.4449767470359802, 0.27159377932548523, -1.166816234588623, -1.252226710319519, -0.7757284045219421, 0.265610933303833], [1.239011526107788, 0.08439398556947708, -0.733564019203186, 2.068077564239502, 0.7658389210700989, -2.429597854614258, 0.6259574294090271, 1.11574125289917, 1.0360976457595825, -0.5827099680900574, -1.6387712955474854, 0.041770175099372864, 0.6230417490005493, 0.5107347369194031, 0.30974385142326355, -0.7572271823883057, 0.06357128173112869, -0.0908389613032341, -0.551916241645813, 1.7818981409072876, -0.34286391735076904, -0.3243288993835449, 1.8203895092010498, -0.4510249197483063, -0.4770224988460541, -0.1829509437084198, -1.3457223176956177, 0.5456275343894958, -0.4376218914985657, 0.14485102891921997, 1.0282936096191406, -0.6867288947105408, -0.5724254250526428, -1.6912389993667603, 0.6811192035675049, -0.5750874280929565, 0.7118764519691467, -0.19357949495315552, -0.6427901387214661, -0.9549718499183655, 1.0946266651153564, 0.5498635768890381, 0.9548963308334351, -0.15797169506549835, -1.4800033569335938, -0.21443970501422882, 0.5532279014587402, 0.6352826952934265, 0.8350695371627808, 1.027267575263977, 0.9047111868858337, 1.6210627555847168, -1.7838870286941528, 0.6620467305183411, 0.6275163888931274, 0.8135183453559875, 0.11769786477088928, -0.7375169992446899, 0.2566777765750885, 0.7859528660774231, 0.11987647414207458, 1.0928168296813965, 0.9399109482765198, -2.0839767456054688], [2.3498482704162598, -1.0708492994308472, 2.4623608589172363, 0.7926816940307617, -0.5858049988746643, 0.3179114758968353, -0.06710135191679001, 0.6199281215667725, -1.3996895551681519, -0.3038055896759033, -2.373401641845703, -0.0018137304577976465, -0.5057306289672852, -1.1171354055404663, -1.0261698961257935, -0.18624024093151093, -0.6032435297966003, -0.17400169372558594, -0.5202615857124329, -0.581221342086792, -0.6659215092658997, 1.0639845132827759, -0.3995739817619324, -0.0237455815076828, 0.7303004860877991, 1.3135446310043335, -1.4243388175964355, -0.3537812829017639, -0.965740442276001, 0.17820216715335846, 0.16999439895153046, -1.679962158203125, -1.0156183242797852, -0.8059147000312805, 0.9065230488777161, -2.142859935760498, 0.5250865817070007, 0.02191033400595188, -0.6009505391120911, 0.15073111653327942, -0.7178676128387451, 1.9166771173477173, 0.08365476876497269, -0.4903925359249115, 1.1148492097854614, 0.07285487651824951, -0.8576450347900391, -0.2296055257320404, 0.405775785446167, 0.08062738925218582, 0.2561505436897278, 0.26827287673950195, 0.7135244607925415, -0.05450987443327904, 1.8493956327438354, 0.5918864011764526, 1.2361071109771729, 0.04546704888343811, -0.6481087803840637, -0.027397265657782555, 0.19679543375968933, -0.5326281189918518, -2.902712345123291, -0.551166832447052], [0.6571003794670105, 0.9516076445579529, -0.7081255912780762, 0.6946509480476379, -0.11901203542947769, -0.45571425557136536, 0.14059719443321228, -0.22661854326725006, -0.213186115026474, -0.4733521044254303, -0.8519110679626465, 3.143766403198242, -0.43265828490257263, -1.5934560298919678, -0.25186729431152344, -0.8862674832344055, 0.27560314536094666, 0.6952568888664246, -0.31996434926986694, 0.7123983502388, -1.1307231187820435, 1.9201085567474365, 0.4347623586654663, 2.023414373397827, -1.6430987119674683, 1.6876749992370605, -0.603007435798645, 0.4601212739944458, 0.11504985392093658, -2.3331620693206787, -0.6533132195472717, -1.1302424669265747, 1.2718693017959595, -1.4425193071365356, -0.0745634138584137, 2.509397268295288, 0.11017163097858429, -2.757603168487549, 0.2488696575164795, 0.01762229949235916, -0.5459121465682983, -1.0588841438293457, -0.5349489450454712, -0.7041156888008118, 1.1483572721481323, 0.7383132576942444, -0.595747709274292, -0.7563234567642212, -0.758194625377655, -0.10435797274112701, -0.8048838376998901, -1.11711847782135, 1.132141351699829, -0.232492133975029, 0.6751916408538818, -2.0760695934295654, 1.9282574653625488, 1.7921422719955444, 0.2754351794719696, -2.5560898780822754, 0.8070176839828491, -0.6118294596672058, 0.550127387046814, 0.21763576567173004], [-1.4503347873687744, -0.11593370884656906, -0.47743460536003113, 0.3524051904678345, -1.3727442026138306, 1.393254280090332, -0.11905152350664139, 1.2917627096176147, 0.5210446715354919, -0.11773208528757095, -0.934425413608551, -0.16670095920562744, -0.8871771097183228, 1.6970772743225098, -0.8113023042678833, 0.04613278806209564, -0.7830929160118103, -0.6320576667785645, 0.16719764471054077, 2.114938974380493, -1.1465184688568115, 0.08030100166797638, 1.0949546098709106, 0.01656132936477661, -0.47267940640449524, 2.8966751098632812, 0.37298583984375, -0.5921692252159119, -1.6615408658981323, -0.9433864951133728, -0.5785303711891174, 0.9783948659896851, 0.035062529146671295, -0.8032646179199219, 1.7297933101654053, 1.1550370454788208, -0.41155683994293213, 0.9132372736930847, 0.04969555512070656, -0.5175139307975769, -1.0658416748046875, -0.20912989974021912, 0.11002882570028305, 0.3161357641220093, 0.17875659465789795, 1.0579993724822998, 1.5154297351837158, -0.02087102085351944, 0.6771761775016785, 1.274680733680725, -0.24154174327850342, -0.7013731598854065, -2.0616202354431152, -0.24207167327404022, -2.1513290405273438, -1.0055196285247803, 0.41423743963241577, 0.8350659608840942, -1.3102165460586548, -2.1053590774536133, -1.380631446838379, 0.3243478834629059, 2.095951795578003, -0.11143551021814346], [-2.0513670444488525, -1.1758959293365479, 1.1898902654647827, 0.4166967272758484, 0.638815701007843, -0.14055192470550537, -0.08746081590652466, -2.0720248222351074, 0.912452220916748, 1.6349561214447021, -0.6527641415596008, -0.7882694602012634, -0.09787412732839584, -0.378527969121933, -1.284877061843872, -1.162109375, 0.23162078857421875, 1.1910109519958496, -0.6086152791976929, -1.5126032829284668, 0.4439091384410858, 1.1310937404632568, -0.7105218768119812, -1.1273638010025024, -0.5661898255348206, -0.2694997191429138, 1.5110726356506348, -1.0620620250701904, 0.3518851101398468, 0.45665988326072693, -0.44298484921455383, 1.7807590961456299, 0.6445503234863281, -0.6059103012084961, -0.28282439708709717, 0.050780102610588074, 1.6694740056991577, -0.315757691860199, 0.7461427450180054, 0.44474464654922485, -1.026462435722351, 0.3434520959854126, -1.0671406984329224, -0.4233205318450928, -0.14164400100708008, 0.6045663952827454, -0.11205337941646576, -0.42172592878341675, -0.5652124285697937, -0.45104923844337463, 0.17213821411132812, -1.3549357652664185, -0.7203165292739868, -0.020635245367884636, -2.486882448196411, 0.6467735171318054, 0.8566967248916626, -0.725279688835144, -0.07952059060335159, -0.09864263981580734, -0.4496613144874573, 0.13944891095161438, -1.1889351606369019, 0.7590968012809753], [-0.6861369609832764, 1.0561244487762451, 2.6870663166046143, 0.26473096013069153, 0.48823845386505127, 0.2995580732822418, -0.11562822014093399, 0.6959853768348694, 1.0416502952575684, -0.5440841913223267, 1.3410097360610962, -0.24565909802913666, 1.4944301843643188, -0.19581827521324158, -2.1069986820220947, -0.41552937030792236, -0.9402577877044678, -0.20668281614780426, 0.8323878049850464, -0.6934300661087036, -0.6725265979766846, 0.15019501745700836, -1.4208316802978516, -0.47103556990623474, 1.3006439208984375, 1.4739469289779663, -0.5705240964889526, -0.3819601237773895, 0.7485178709030151, 1.2367621660232544, -0.3563151955604553, -0.8675207495689392, -0.7140162587165833, 1.2010722160339355, 0.5514052510261536, -0.7877522706985474, 1.657217025756836, -0.28888264298439026, -0.7833958864212036, 1.0739519596099854, 1.0733839273452759, -0.658380925655365, -0.5101917386054993, 0.6535712480545044, -0.2902759313583374, 1.2732518911361694, 0.8794219493865967, -0.4909162223339081, -2.785942792892456, 0.10056933015584946, -0.01538501214236021, 2.225454330444336, -1.002367377281189, -0.10403374582529068, 1.6493291854858398, -0.1668483316898346, 0.9911093711853027, 0.2756485641002655, -0.01437193900346756, 0.6910043954849243, -0.3619421422481537, -0.8401368260383606, 0.5256587862968445, -0.28461506962776184], [0.4774847626686096, 1.3869210481643677, 0.8919420838356018, -1.107468605041504, -0.8812538981437683, -1.3305654525756836, -0.5001844763755798, -0.6995905637741089, 0.5013925433158875, 1.2758833169937134, 0.46279457211494446, -2.0223660469055176, -0.23175691068172455, 0.07605810463428497, 1.3366448879241943, 0.4811779260635376, 0.8144302368164062, 0.2928929626941681, -1.4625734090805054, 0.5484941005706787, 1.0686147212982178, -1.0800366401672363, 1.7852076292037964, 2.3322341442108154, -0.8176872730255127, -0.5964502096176147, -1.0260601043701172, 0.3039773106575012, -0.8102144598960876, 1.7806837558746338, -0.5206454396247864, 0.006084630265831947, 0.7532508969306946, -0.3666476309299469, 1.661636471748352, 0.004683622159063816, 0.24243110418319702, 0.7920036911964417, -1.2625850439071655, 0.21334950625896454, -0.5892475843429565, 0.2808832824230194, -1.071326732635498, 1.4275027513504028, 0.6683521270751953, -0.32268622517585754, -0.08973655849695206, -1.6086349487304688, 0.48136091232299805, -1.1775418519973755, 0.873238742351532, -0.6808737516403198, 1.6913580894470215, -0.265873521566391, -0.5241625308990479, 1.1472299098968506, -0.12409181892871857, 2.026481866836548, 2.0486998558044434, -0.5572580099105835, 1.2261829376220703, 0.6659159064292908, 0.21805201470851898, -1.8022713661193848], [-0.05654536932706833, -0.4776655435562134, 1.1926982402801514, 1.2784969806671143, -1.0840094089508057, -1.352274775505066, 0.24642325937747955, -0.38440269231796265, 0.9618211388587952, -0.50386643409729, 1.3858751058578491, 1.7485994100570679, -1.2116349935531616, -0.9311841726303101, 0.16196748614311218, -0.2306334525346756, 1.954773187637329, -0.0922708585858345, -0.18976104259490967, -1.5163140296936035, 0.5690574645996094, -0.9476891756057739, -0.6061000227928162, -1.9566380977630615, -0.2070157378911972, 0.15937048196792603, 0.1928853690624237, -1.5225390195846558, 0.1829889863729477, 0.3792528510093689, 0.7313146591186523, -1.4871175289154053, 0.23890957236289978, 0.2612496018409729, -0.0760100781917572, 0.34789663553237915, -1.0448111295700073, -0.8278177976608276, 2.286282539367676, 1.1180987358093262, 0.8371188640594482, 0.4227409362792969, -0.3331139087677002, -0.721198558807373, -1.3401923179626465, -0.16838869452476501, -1.6816905736923218, -0.00589874479919672, -0.8009464144706726, -0.38837796449661255, -0.5258849263191223, 1.8006924390792847, 1.943821668624878, -1.1254684925079346, 2.2226641178131104, 0.9667279720306396, 0.632936954498291, 1.1696771383285522, -2.4694859981536865, 0.3038582503795624, 1.8215856552124023, -0.397116482257843, 1.3514695167541504, -0.5575533509254456], [-0.9451057314872742, 0.5276089906692505, -0.281904399394989, -0.5971977114677429, -0.1940467655658722, 0.19336630403995514, 0.8282409906387329, 0.0034860463347285986, 0.19435609877109528, -0.08308684080839157, -0.6756108403205872, -0.7915363311767578, -0.34897059202194214, -0.6120470762252808, -0.4492850601673126, 0.6536174416542053, 0.13391342759132385, 0.11103737354278564, -0.8472356200218201, -0.40043285489082336, -0.3318954408168793, -0.644652247428894, 0.752005934715271, 1.8611152172088623, 1.065205693244934, -0.9258942604064941, -0.3567332923412323, 0.6750712990760803, -0.25203773379325867, 0.124990314245224, -1.2066665887832642, -1.2169902324676514, 0.5921534895896912, -0.6818658113479614, 0.6822362542152405, 0.9620978832244873, -0.7281033992767334, -0.8037620782852173, 1.6192330121994019, -0.106594517827034, 0.44247353076934814, 0.42737066745758057, -0.11280351132154465, 0.12017311900854111, 0.7475338578224182, -0.13255898654460907, -1.7780025005340576, -0.37565070390701294, -0.5137608051300049, -0.42468252778053284, 1.0375975370407104, -1.616410255432129, 0.6744170188903809, -1.2442947626113892, -0.5670765042304993, 0.5068130493164062, -0.8604033589363098, 1.297218680381775, 0.9931066036224365, -0.3269566297531128, -0.34139567613601685, -0.4370616376399994, -2.5845959186553955, -0.7899416089057922], [-0.7233834862709045, 0.10579467564821243, 0.6232095956802368, -3.34254789352417, -0.8290130496025085, 0.667599618434906, 0.6080718636512756, 1.4520139694213867, -0.5872042179107666, -1.276552677154541, -0.04261567443609238, 0.5261742472648621, -1.069409966468811, -0.9480657577514648, -1.2489149570465088, 0.5761538743972778, -1.2343143224716187, 0.07732219994068146, -0.0430561788380146, -0.28202009201049805, 0.3582209050655365, -0.025060798972845078, -1.3215785026550293, -0.9385651350021362, 0.32126176357269287, 1.7163664102554321, 0.7268438935279846, 0.0422331802546978, 0.22044673562049866, -0.8781000375747681, 1.420742392539978, 1.1155004501342773, -2.2221121788024902, 0.6948548555374146, -1.7874395847320557, -0.04900744557380676, 1.2307541370391846, -0.8020151853561401, 0.07729672640562057, -0.22445333003997803, -3.2846643924713135, 2.083717107772827, 0.5839570760726929, -2.6031744480133057, 1.4761407375335693, -0.6968451738357544, -0.5695924162864685, -0.5524478554725647, 0.9880194067955017, 0.49623537063598633, -0.9830886125564575, -0.09211330115795135, -1.1616291999816895, 1.1547589302062988, 0.27897292375564575, 0.9197271466255188, -1.3264442682266235, -0.010691859759390354, 1.8354469537734985, -0.5726884603500366, -0.11663331091403961, -0.5527477860450745, -0.7236055731773376, 0.11061165481805801], [-0.21891574561595917, -0.8760361075401306, -0.8337336778640747, 0.5058481097221375, 1.4370992183685303, 0.36911630630493164, 0.48022300004959106, -0.6535972356796265, -0.34790775179862976, -0.8374881148338318, 0.02206178940832615, 0.013182888738811016, 1.2302244901657104, 1.3357030153274536, -0.4501102864742279, 0.7637640833854675, -0.15581750869750977, -1.1720982789993286, -0.5930302739143372, 0.3229505121707916, 0.2740674912929535, 0.6931359171867371, -0.7698983550071716, -0.22606179118156433, -1.8208484649658203, 0.9152842164039612, 3.4374983310699463, 0.3508489429950714, -2.1545486450195312, 0.02427651360630989, -0.0293339304625988, 0.8218543529510498, 0.609031617641449, 1.5338897705078125, 1.2194589376449585, 0.4631563723087311, -0.6331737041473389, -0.6024254560470581, 0.19330397248268127, -0.42363524436950684, 1.1290391683578491, 0.7487088441848755, 0.8166717290878296, -2.481024742126465, -0.9183474183082581, 1.1916863918304443, 0.1680484414100647, 0.15365883708000183, 0.8641029000282288, -0.5934807658195496, -0.6296699047088623, -0.36915087699890137, -1.213889479637146, -0.24535802006721497, -2.0327625274658203, 1.4960826635360718, 1.768652081489563, 0.729274332523346, 0.20396143198013306, -1.9716213941574097, 2.104060649871826, 0.6950942277908325, -0.7469016909599304, -0.8515223860740662]], [[0.215719535946846, -0.7890348434448242, 0.45944616198539734, 0.9960584044456482, -1.2871137857437134, -1.545632243156433, -0.6743455529212952, 0.9968156814575195, 0.2716684341430664, 0.1493447870016098, -0.19335272908210754, 1.206682562828064, -0.9945492744445801, 1.3325273990631104, -0.7734037041664124, -0.421720951795578, 0.4113522469997406, 1.3482378721237183, 0.20328286290168762, -1.3184165954589844, 0.08138656616210938, -1.6370784044265747, 0.8903610110282898, 0.13461121916770935, -0.08006524294614792, 0.6031178832054138, -1.1619292497634888, -1.9120562076568604, -0.35646969079971313, -0.42201247811317444, -1.9919483661651611, 0.14886035025119781, 0.973504900932312, 1.5702497959136963, 0.6665960550308228, 0.011287190951406956, -2.704575538635254, -2.480792284011841, 0.9032522439956665, 1.4930813312530518, -0.7116197347640991, 0.11985185742378235, 0.605167806148529, 0.3636355400085449, 0.8009308576583862, -0.7459663152694702, -1.7445483207702637, -0.2033787965774536, -0.025012679398059845, -2.7039244174957275, -0.12161006033420563, 2.755110740661621, 0.8847886919975281, -0.24220021069049835, 0.6795443296432495, 0.656981348991394, 0.13499975204467773, -0.26785463094711304, -1.3704311847686768, -1.2016208171844482, 0.6364750862121582, -0.6345766186714172, -0.38839712738990784, 0.7103890180587769], [0.6898196935653687, -0.5182366967201233, -1.6068466901779175, -0.5054689645767212, -0.9463163018226624, 0.6288233399391174, -1.7542431354522705, 0.09206689894199371, 0.4256588816642761, -1.945345401763916, 0.29422688484191895, 0.24584831297397614, 1.6477679014205933, 0.542008638381958, 0.8491106629371643, 0.19968603551387787, 1.5556334257125854, -0.5951604247093201, 1.8135415315628052, 1.1261776685714722, -1.6873834133148193, -0.6011731028556824, 1.4067797660827637, 1.140290379524231, -0.2254898101091385, -0.3412075340747833, 0.7605263590812683, -1.5522981882095337, 0.026583373546600342, 1.407947063446045, -1.4376176595687866, 1.7180759906768799, 1.8576717376708984, -0.6231099963188171, -1.1760344505310059, -0.025834767147898674, 0.47607776522636414, 0.8948854804039001, -0.343863308429718, -1.1520988941192627, 1.2355684041976929, -0.7279402017593384, -1.1200921535491943, -0.8507999777793884, 0.6682566404342651, -0.5586385726928711, -0.5070435404777527, 0.981611967086792, -1.0465402603149414, -2.5902819633483887, 0.33759018778800964, 0.4032415449619293, 0.4132152497768402, -0.650031328201294, -1.4146579504013062, -0.47281256318092346, -1.1700899600982666, 0.9550498723983765, 0.31329578161239624, 1.9712265729904175, 0.7899218201637268, -0.22259646654129028, 1.4386425018310547, -0.8279028534889221], [1.196254849433899, 1.1264411211013794, -0.26473698019981384, -0.7888810634613037, 0.020853696390986443, -0.45334210991859436, 0.0996149480342865, 0.5201098322868347, 0.3421051800251007, 0.13095925748348236, 0.9654352068901062, 0.3531239330768585, 1.7259016036987305, -1.620312213897705, 0.2662351727485657, -0.5921801328659058, 1.2262197732925415, 0.12452042102813721, 1.2975975275039673, 1.6699188947677612, -0.1101493090391159, 1.1577295064926147, 0.9623712301254272, -0.6914200782775879, -0.564709484577179, 0.8470078706741333, -1.9299921989440918, -0.4031526744365692, -0.5591127276420593, -0.1526847779750824, 0.016906458884477615, 2.030733108520508, -0.40436941385269165, -1.2529168128967285, 0.05146360769867897, -0.9524176120758057, 0.9264519810676575, 0.8744187951087952, -1.4202691316604614, -1.3746707439422607, 0.30312424898147583, -0.48644018173217773, -0.05058625712990761, 1.5220937728881836, -0.10497400909662247, 0.4083549380302429, -1.2929778099060059, -1.444392204284668, -1.0376129150390625, -0.6254894137382507, 0.5712074041366577, 0.9332399368286133, 1.7052420377731323, -0.895646333694458, 0.8229856491088867, -0.781237781047821, -1.6984175443649292, 1.5907405614852905, 0.5636669397354126, -0.7735167145729065, -0.22781804203987122, 0.8484704494476318, 1.9563467502593994, 1.0391325950622559], [0.8286913633346558, -0.7347940802574158, -1.615257978439331, -0.006473924964666367, -1.3265957832336426, -0.4333399534225464, 1.3843841552734375, 0.282093346118927, -1.0594866275787354, -1.7616956233978271, -1.0979303121566772, 0.6807953119277954, -2.4529407024383545, -1.1823585033416748, 0.5488222241401672, -0.6138034462928772, -0.05295569449663162, -1.1356760263442993, 0.9453004598617554, 2.344595432281494, 0.5353612899780273, 1.532835602760315, -0.3714263439178467, 0.41627037525177, 0.04970235377550125, -1.7143573760986328, -0.09267795830965042, -0.037677861750125885, -0.4816981852054596, -0.3135668635368347, -1.823951244354248, 0.6384023427963257, -0.7880344986915588, 0.23204068839550018, -0.5069634914398193, 0.9938857555389404, 0.08191949129104614, 0.7019092440605164, -1.1911687850952148, 0.6750460863113403, 0.47283580899238586, -0.1456248015165329, -0.4782790243625641, 0.3552609980106354, 0.5156674981117249, -0.030132507905364037, 0.13364839553833008, -0.405962735414505, -0.68131023645401, -1.350484013557434, 1.042153000831604, 0.4190343916416168, 0.16358882188796997, -0.3169989585876465, -0.19954873621463776, -0.6331744194030762, 0.8837574124336243, 0.24794241786003113, 0.09455978125333786, -0.21314017474651337, -0.8213616609573364, 1.378279447555542, -0.14551204442977905, 0.9328216314315796], [-0.62176114320755, 0.1986643373966217, 1.1359578371047974, -1.3116897344589233, -0.8056994080543518, -0.9799292087554932, 0.3439631760120392, -1.7288790941238403, -0.4941812753677368, -1.477062702178955, -1.7083483934402466, 2.2749948501586914, -0.32272735238075256, 1.2523545026779175, 1.1838099956512451, 1.569666862487793, -0.139162078499794, -1.1326251029968262, 2.260035991668701, -1.6344363689422607, -1.8799909353256226, 0.6670931577682495, 0.9103142023086548, 0.8894047737121582, -0.29963478446006775, -1.3832966089248657, 0.2518208920955658, 1.0611765384674072, -0.08162710070610046, 1.4998586177825928, -0.07158855348825455, 0.9284087419509888, -0.7146397829055786, 0.4521181285381317, 1.512770414352417, 1.3887156248092651, 0.7150481939315796, -0.05897825211286545, -0.6227583289146423, -0.001390890683978796, 1.102766990661621, 0.8784970045089722, -1.0547735691070557, 1.3136777877807617, 1.5840703248977661, -0.6692838072776794, 0.515053391456604, -0.9876570701599121, 0.17001758515834808, -0.062388282269239426, 0.35507193207740784, -0.021324768662452698, -0.45788511633872986, 0.4718484878540039, -2.468313455581665, -1.1020554304122925, 0.013128926046192646, -0.09577500075101852, 0.5765439867973328, -0.9563531875610352, -1.4600740671157837, -0.03059164434671402, -0.5708193182945251, -2.019652843475342], [0.9232614636421204, 0.40748193860054016, -0.13205529749393463, 2.5841548442840576, -0.9323763251304626, 0.14126801490783691, 0.6972410678863525, -0.019493380561470985, -0.5074684619903564, 0.05821641534566879, 0.3699660897254944, 0.8616219162940979, -0.5769234299659729, -0.4470294117927551, -0.851676881313324, -0.5730496644973755, -0.5430657267570496, -0.6752426028251648, 1.8789315223693848, -0.601634681224823, -0.5553030371665955, 2.284723997116089, 0.839337170124054, -2.7908899784088135, -0.06614029407501221, -0.29702994227409363, -1.5700162649154663, -1.3409830331802368, 0.6956264972686768, 0.3860680162906647, 0.051935888826847076, -0.8930606245994568, 1.2549015283584595, 0.2207430899143219, 0.005342231132090092, 0.059168629348278046, -1.670541524887085, 0.3807331621646881, 1.4561846256256104, 0.34612277150154114, -0.507881224155426, 1.1356773376464844, -1.3353190422058105, -0.6403961777687073, 0.27470630407333374, 0.5331928133964539, 0.3766433894634247, -1.6821831464767456, 0.5091484189033508, 0.34131935238838196, -0.7811291813850403, 1.6300379037857056, -0.2739735543727875, -0.14292462170124054, 0.5675291419029236, 0.3372799754142761, 0.3434775173664093, -0.6091005802154541, 0.2305477261543274, -0.0070664179511368275, -1.1855071783065796, -1.5954086780548096, 2.2633488178253174, -1.1375826597213745], [-0.8352458477020264, 0.6895403861999512, 0.21924564242362976, -1.7912381887435913, 0.6076593995094299, 0.744586169719696, 0.005988870281726122, -0.4493226110935211, -0.035404354333877563, -0.9513779282569885, -1.239656925201416, 0.38977909088134766, 1.8019905090332031, 1.1065620183944702, 0.42001834511756897, 0.7838947176933289, 0.5054168105125427, 0.9687461256980896, 0.9649083018302917, -1.19186270236969, -0.7187884449958801, 1.5398460626602173, -0.85878586769104, -1.8298577070236206, 1.2785967588424683, 0.09199512004852295, 0.5409001708030701, 1.2822494506835938, -0.9324820637702942, 0.20751549303531647, 0.3195757567882538, -0.24133028090000153, -0.09103785455226898, -0.28334543108940125, 0.5140523314476013, -0.7393580675125122, -0.10627766698598862, -1.1299480199813843, 0.12714357674121857, 0.9476136565208435, -0.7605385184288025, -0.2523583471775055, -1.060828447341919, 1.8155885934829712, -0.019391732290387154, 0.4182718098163605, 2.131227731704712, -0.25396236777305603, -0.09531241655349731, -0.7413858771324158, -1.2621725797653198, 1.5981435775756836, 0.2761390805244446, 0.08378727734088898, 0.4385521709918976, -0.6730919480323792, 0.2765045166015625, 0.5380395650863647, 1.0719013214111328, -0.10434254258871078, -0.4210846722126007, 0.11686112731695175, 0.7484068274497986, 1.0358035564422607], [-0.290200799703598, -1.392781138420105, 0.5661929845809937, -0.6166269183158875, 0.0653839036822319, -0.7319149971008301, 0.0123514449223876, -1.2240556478500366, 1.8576774597167969, -0.08712191134691238, 0.36802250146865845, 0.5836265683174133, -0.599924623966217, -0.9084541201591492, -0.5517690181732178, -1.2804356813430786, -0.4839371144771576, -0.13571538031101227, 1.0018854141235352, 1.4212778806686401, -1.4169940948486328, 0.3891146183013916, 0.563980758190155, 0.42916014790534973, 0.850999653339386, 1.3724244832992554, 0.5843043327331543, -0.5991141200065613, -0.23706476390361786, -0.3004901111125946, 1.7593046426773071, -0.49212607741355896, -0.8056045174598694, 0.5157224535942078, 1.1607221364974976, 1.5162609815597534, -0.2659449577331543, -1.212491512298584, 0.194652259349823, 0.8241933584213257, 0.5178263187408447, 1.681849718093872, 0.5093337297439575, 1.019230842590332, 0.8537811040878296, 0.009187937714159489, -0.6005809307098389, -0.9390529990196228, -1.5198556184768677, -0.5399970412254333, -0.7366099953651428, -1.5985201597213745, 0.17771707475185394, 0.17498652637004852, -1.8447539806365967, -0.4632081687450409, -0.9064469337463379, 0.26871350407600403, -0.24723869562149048, -1.1145049333572388, 0.22349804639816284, -0.6247085928916931, 0.8020216226577759, 0.4552645683288574], [0.7740017771720886, 1.3758983612060547, -1.010311245918274, 2.244762659072876, 0.21196608245372772, -0.6677922606468201, -0.8562031388282776, 0.5832969546318054, 1.8201786279678345, -0.42257246375083923, -1.089469313621521, 1.4492202997207642, -1.0714870691299438, 0.5839068293571472, 0.7351388931274414, -0.016495466232299805, -1.3410420417785645, 0.756488561630249, 0.5641587972640991, -0.0029218322597444057, 1.004654049873352, -0.4106641113758087, -1.526393175125122, -2.2575302124023438, 0.23859737813472748, -0.5562986135482788, 0.6953704357147217, 0.3404626250267029, 0.15129968523979187, -0.6203209161758423, 0.063806913793087, 0.9556921720504761, -1.0382581949234009, 1.3428839445114136, -0.1775844693183899, 0.6758065819740295, -0.6917272806167603, 0.8570429682731628, -0.1043778732419014, -0.001517641358077526, 0.33716660737991333, 1.7465507984161377, 0.7136356234550476, 0.2239709049463272, -1.1539636850357056, 0.012101172469556332, -1.238045573234558, 0.3685716986656189, -0.36055371165275574, 0.031235797330737114, -0.8264997005462646, -0.01167932990938425, 0.2668876051902771, 1.6840014457702637, 0.5267770290374756, 0.7078707814216614, -0.9992595314979553, 0.8918015360832214, -0.5841625928878784, -0.9892972111701965, -0.3919677436351776, 0.23650150001049042, -0.44198405742645264, -0.5960873365402222], [0.5667828321456909, -0.871913492679596, -0.380439817905426, -0.587908923625946, -0.013651865534484386, 0.13961847126483917, -0.836245596408844, 0.9898120164871216, -2.8554301261901855, 0.33680614829063416, 1.2173455953598022, 1.5702567100524902, 0.5069186687469482, 0.5804793238639832, -0.4200607240200043, -0.40663957595825195, -1.0223637819290161, -0.568574070930481, -1.6494179964065552, -2.1037511825561523, 1.0410175323486328, 0.31300222873687744, 1.2333980798721313, -0.3030479848384857, -1.350504755973816, 0.2949436604976654, 1.1531990766525269, -2.011070966720581, -0.38047170639038086, 0.054904405027627945, 2.128750801086426, 0.22440873086452484, -1.2694746255874634, -1.209602952003479, -1.146852731704712, -0.9923461675643921, 0.5868125557899475, -0.23126612603664398, 0.056178655475378036, -1.5770328044891357, 2.105036735534668, -0.7164774537086487, -1.5560753345489502, -0.512919545173645, 0.5452418327331543, -0.4897894263267517, -0.2233441174030304, 2.0557546615600586, -1.0299097299575806, 0.6280494332313538, -2.668330669403076, 0.06683304160833359, -0.6971495747566223, 0.7145875096321106, -0.05761535465717316, -0.8951294422149658, -0.12770245969295502, -0.07688138633966446, -0.40133944153785706, -0.914107620716095, -1.3307371139526367, -0.9239300489425659, -0.5489113330841064, -1.0649186372756958], [0.4963129162788391, -1.561405897140503, -2.066798686981201, -1.492009162902832, -1.1996415853500366, -0.5139029622077942, 0.4937843084335327, -0.260924756526947, 0.7878021597862244, -0.11639469861984253, 0.07733634859323502, 1.099268913269043, 0.002393025206401944, 0.35443392395973206, -0.5562495589256287, -1.1504085063934326, -0.5187245011329651, 1.5052831172943115, 0.12850889563560486, -2.5081727504730225, -1.362428903579712, 0.3671036660671234, -0.7674247622489929, 0.6514360308647156, 0.23178640007972717, -0.4764327108860016, 1.6697980165481567, -0.34524449706077576, 2.0695712566375732, 0.3641725480556488, 0.5600192546844482, 0.25023820996284485, -1.0634558200836182, -0.35904693603515625, 0.3554201126098633, -1.0170890092849731, 0.9945668578147888, 0.08421322703361511, 0.09665525704622269, -0.09078460186719894, -2.5115504264831543, 0.3320245146751404, 0.8143190145492554, 2.077313184738159, 0.7856117486953735, -0.047735102474689484, 0.21725349128246307, -0.43049174547195435, -0.15159983932971954, -1.730840802192688, 0.09637484699487686, 1.4444059133529663, 0.2682484984397888, -1.032415509223938, -0.5011813044548035, 1.0629050731658936, 1.2705570459365845, -1.402072548866272, -0.9204955101013184, -0.16330304741859436, 0.6171742677688599, 0.14191630482673645, 0.2636215090751648, 0.9756128191947937], [0.6684443354606628, -1.3206429481506348, 0.49622589349746704, -0.43643462657928467, -0.3996821343898773, 0.07055272161960602, -0.14079627394676208, -0.14497819542884827, 0.9118967652320862, 2.021278142929077, -0.31508854031562805, 0.42562732100486755, -0.8233931660652161, -1.307288646697998, -1.3422399759292603, -1.473580002784729, -0.11141392588615417, -0.24878768622875214, 1.2382745742797852, 0.31413745880126953, 0.36372390389442444, -0.4757508933544159, -0.5638067722320557, 0.8231837153434753, 0.2501618564128876, 0.6020488739013672, 0.7098177075386047, -1.0660945177078247, -0.2369580715894699, 1.6201928853988647, 0.7105090022087097, 1.9236706495285034, -0.04942002147436142, 0.6037179827690125, 1.575691819190979, 3.098357915878296, 0.6821735501289368, -0.6302111744880676, 0.0034660242963582277, -0.6857143640518188, -0.4619493782520294, 0.08819479495286942, -0.5416510105133057, 0.41368597745895386, -1.6099778413772583, 0.16098198294639587, -0.8082388639450073, -0.3404647707939148, 1.4887067079544067, -0.5158957242965698, 0.13361112773418427, 0.6512170433998108, 1.0306516885757446, 0.9626544117927551, -0.37627479434013367, -0.67633455991745, 0.9881640076637268, 1.333283543586731, -1.118929386138916, 0.9854008555412292, -0.0168533343821764, 0.5753874778747559, -0.7428656220436096, 0.9817422032356262], [1.0569665431976318, -2.0024991035461426, 1.508378028869629, 0.28081706166267395, -1.0761548280715942, 0.9623749256134033, -1.0347895622253418, 0.788589358329773, 1.4554024934768677, -0.48890239000320435, -0.7082083821296692, -1.5783082246780396, -0.6206771731376648, 1.7796722650527954, -1.094179630279541, 1.4440689086914062, 0.8876802921295166, -0.37380409240722656, 0.6375541687011719, 0.103664830327034, 0.22190959751605988, 0.7557260990142822, 0.5829514265060425, -0.5211575627326965, 0.19264675676822662, 0.24666668474674225, -0.17053650319576263, -0.6662943363189697, -0.09407810866832733, -0.08009974658489227, 0.11214112490415573, -1.2102422714233398, -1.9722485542297363, 0.7821441292762756, 0.17914681136608124, -0.975715696811676, 0.8059186339378357, -0.3310781419277191, -0.8140372633934021, 1.292372226715088, 1.0878539085388184, -0.3116684854030609, 0.9851370453834534, 0.6020064353942871, -1.4095549583435059, -1.5808022022247314, 0.16800585389137268, -0.5817394256591797, -1.0385444164276123, -0.27645811438560486, 0.2438696026802063, 0.7480424046516418, -0.2147715836763382, -0.8249471187591553, 0.894482433795929, -1.2201793193817139, -1.230031967163086, -0.5986416339874268, -0.6802225708961487, 0.30633261799812317, -0.5755220055580139, 0.8812527060508728, -0.7378180623054504, -0.6719323396682739], [1.7047152519226074, -0.3520689010620117, -0.602418065071106, 0.29352790117263794, 1.1967328786849976, 1.2692457437515259, -0.8181480765342712, -1.7465485334396362, -1.1991254091262817, -0.46909138560295105, 1.0246649980545044, 1.289578914642334, -0.7801135182380676, -1.0157361030578613, 1.0710642337799072, -0.13918302953243256, -0.13066576421260834, 1.4851638078689575, 0.693018913269043, -1.3669573068618774, 0.3423025906085968, -0.006052742246538401, 0.9004168510437012, -1.4527206420898438, -0.22787746787071228, -0.8512318730354309, 0.04989797621965408, -1.026625394821167, 0.7068508267402649, -1.082442045211792, 1.0134449005126953, 1.0995088815689087, -0.14998850226402283, 1.3312879800796509, 1.338014841079712, -0.16784027218818665, -0.9498894810676575, -0.6024373173713684, -0.6238706707954407, 1.0353435277938843, 0.7977229356765747, 0.2786141633987427, 0.7325548529624939, 0.8908532857894897, -0.7932363152503967, -0.47650113701820374, -1.3007442951202393, 0.7603474259376526, 0.5230316519737244, 0.8778253197669983, 0.2476375848054886, 0.2967016398906708, -0.6265799403190613, -0.7391803860664368, 1.441675066947937, -2.0677647590637207, 1.2472718954086304, -1.22791588306427, -0.6884373426437378, 1.0220812559127808, -0.7787095904350281, -1.835768222808838, 0.6508118510246277, 0.8719595074653625], [0.4449690580368042, -1.3926411867141724, 0.27443167567253113, -0.06139953434467316, -0.31569039821624756, 0.2801777720451355, -1.024943470954895, 1.3174673318862915, -0.6301632523536682, 0.1548791378736496, 0.4129609167575836, -0.07001426070928574, 0.7945465445518494, 0.041081104427576065, 0.5517203211784363, -1.1782623529434204, 0.46982088685035706, 0.004740480333566666, -1.2917652130126953, -0.023715466260910034, -0.6925429701805115, -0.561538577079773, -1.506384015083313, -1.0710582733154297, -0.8297767043113708, -0.1290997415781021, 0.027186434715986252, -0.29170915484428406, -0.9366260170936584, 0.013616473414003849, 1.1745659112930298, 0.6639390587806702, 0.2909291386604309, 0.3353727459907532, -0.19486062228679657, 0.09311743825674057, 0.36810216307640076, 0.31239694356918335, 0.6737570762634277, 1.076369285583496, 1.1380754709243774, 0.524297833442688, -0.24860087037086487, -1.0942487716674805, -1.6498135328292847, 1.022222876548767, 0.6211788058280945, -0.07957213371992111, -1.0141019821166992, 0.29273414611816406, 1.387865424156189, 1.4318499565124512, 2.3910272121429443, 0.9516160488128662, 0.9512190222740173, 1.3018566370010376, -1.5704575777053833, 0.8587192893028259, -0.6908383369445801, 0.42054301500320435, -1.5439469814300537, -1.0254284143447876, -0.9492470026016235, -0.5431014895439148], [0.7233948111534119, -1.0671905279159546, 0.7102158665657043, 1.0388729572296143, -1.1404279470443726, 0.9865483045578003, 1.7776020765304565, -0.6226539015769958, 0.6478496789932251, 1.480932354927063, -0.5868710875511169, -0.6231276392936707, -0.1307341307401657, -0.13755415380001068, -0.9965152144432068, -0.3977276086807251, -0.9202225804328918, 0.557403564453125, 2.3761556148529053, 1.0811265707015991, -1.3868377208709717, 1.228020191192627, -0.935524046421051, 0.5965280532836914, -0.279348760843277, 1.4463459253311157, 0.06359265744686127, 0.12031606584787369, 0.015882888808846474, -0.11934179067611694, 0.3579094707965851, 1.8525935411453247, 1.6328771114349365, 0.78110271692276, 0.12531240284442902, -0.05019533634185791, 1.4214770793914795, -0.8233321309089661, -0.9639562368392944, -0.643222987651825, 0.3277381360530853, 0.34441307187080383, 0.4474184811115265, -0.21767795085906982, -0.7044922113418579, -2.061532497406006, 0.3853450119495392, -0.7282955050468445, 1.0543184280395508, -1.117812991142273, 0.12191388010978699, 0.5444864630699158, 0.357079416513443, 0.770431399345398, 0.9263312816619873, 0.3274388909339905, 0.9055979251861572, -0.01342124305665493, -0.0698675736784935, 2.6798934936523438, -0.050890929996967316, -0.25570249557495117, -1.833501935005188, -0.9767597913742065]], [[0.038281477987766266, -0.7735971212387085, 0.021758724004030228, -1.290818214416504, 1.223110318183899, 0.15264274179935455, -0.4423760771751404, -0.4224531948566437, -0.24355284869670868, 0.7376647591590881, -0.6746431589126587, 0.3806312680244446, -0.32882457971572876, -1.300124168395996, -0.4380318820476532, -0.0031836088746786118, 0.4442856013774872, 1.5505515336990356, -1.0369185209274292, -0.6713353991508484, 0.47545769810676575, -0.10785908252000809, -1.5473101139068604, 0.8536614179611206, -1.1364012956619263, -1.4680951833724976, -1.0881061553955078, -1.9614566564559937, -0.8376564383506775, 0.3315235376358032, 1.0519400835037231, -0.45519036054611206, -0.7931910753250122, 0.02703843265771866, -0.559280276298523, 2.75072979927063, 1.668297529220581, 0.5324532389640808, -0.2325395792722702, 0.5093146562576294, 0.8449515104293823, -0.15920484066009521, 0.8561148643493652, 2.8641560077667236, -2.0386078357696533, 0.33423271775245667, 0.9835999011993408, -0.7014506459236145, -1.048711895942688, 0.4508914053440094, -0.09566190093755722, -0.35191449522972107, -1.124457597732544, 1.5770654678344727, -0.2669842541217804, 0.8729398250579834, 0.05067698657512665, -1.2834104299545288, 0.01779438555240631, -1.8440262079238892, 0.5364727973937988, -0.0008468187297694385, -1.6429386138916016, 0.515324592590332], [1.4200340509414673, -0.8402597308158875, -1.991278052330017, -1.884737253189087, -0.6473025679588318, 1.5753830671310425, -1.2744665145874023, 1.8963240385055542, -1.2016324996948242, 0.11579283326864243, -1.6344393491744995, 0.7476978898048401, -0.29614976048469543, 1.233794927597046, 0.103936105966568, 0.4748380780220032, 0.7171542048454285, 0.459953248500824, 0.3818747103214264, -0.23145581781864166, -0.15102888643741608, -0.6347934603691101, 1.3002485036849976, -0.4673907160758972, -2.1454646587371826, 0.40371066331863403, 0.9520857930183411, -0.6246804594993591, -0.08968441188335419, -0.448149710893631, 0.6055550575256348, 1.4635823965072632, 0.1575136035680771, -0.07249808311462402, 1.0814580917358398, 0.5565301775932312, -0.4168778657913208, -0.022524289786815643, 1.6593611240386963, 0.24149754643440247, 0.4873451888561249, -0.951802670955658, 1.4929081201553345, -0.47988244891166687, 0.12101221829652786, 0.26706990599632263, -0.8128165006637573, -1.0834583044052124, 1.2434717416763306, 0.0760277584195137, 1.0035152435302734, 2.4543447494506836, -0.6869756579399109, 0.09037258476018906, 0.17637589573860168, -1.612082839012146, -0.7915107607841492, 2.0376086235046387, 0.07485342025756836, 1.1731663942337036, 2.0016653537750244, -0.3014466464519501, 1.6856789588928223, 1.1270383596420288], [-0.3084764778614044, 0.5642104148864746, 1.0287328958511353, -0.17441754043102264, 1.0070695877075195, 0.8592528104782104, -2.676353931427002, 0.3203377425670624, -0.6908507347106934, 0.7965518236160278, -0.03546055778861046, 0.9542853236198425, -0.3359668254852295, -1.1718419790267944, 1.7823712825775146, -0.3133690059185028, 1.029672622680664, -0.09886310249567032, -0.9996832609176636, -1.0366504192352295, 1.3897426128387451, 0.4461825489997864, 0.2595154047012329, -1.7561167478561401, 0.1688583642244339, -0.35086673498153687, 0.3951159119606018, -0.020496077835559845, -0.2854534983634949, 0.056494735181331635, -0.4297613203525543, 1.209861397743225, -0.5795184969902039, -0.4363517761230469, 1.286057710647583, 0.7864657044410706, 0.1788426637649536, 0.2059105932712555, 0.7374028563499451, -0.015345445834100246, -1.3074350357055664, 1.2314364910125732, -0.16548936069011688, -1.9827580451965332, -1.0510029792785645, -0.7575930953025818, -0.15954189002513885, 1.5111451148986816, -0.21218881011009216, -0.2237548679113388, -1.3096826076507568, 1.127375841140747, 0.043639831244945526, -1.0850536823272705, 0.7040722966194153, 1.5840582847595215, -0.3931346535682678, -0.17784075438976288, 1.0151631832122803, 2.2618961334228516, -1.3404477834701538, 1.450069785118103, -0.5893085598945618, -2.298924207687378], [0.8174931406974792, -1.7727570533752441, -0.8952125906944275, 2.0874528884887695, -0.8671852946281433, 1.1222206354141235, -1.2108792066574097, -0.3632550537586212, 0.430969774723053, -2.281891345977783, -0.6079671382904053, 1.8516777753829956, -0.7515299320220947, -0.26244089007377625, 0.018632451072335243, -0.6072034239768982, 2.066802978515625, -1.7029403448104858, 1.4047539234161377, 0.87850022315979, -0.017252996563911438, 1.2333168983459473, -1.5238844156265259, 0.11439212411642075, 1.006446361541748, 0.4349026083946228, -0.9236762523651123, 0.32567667961120605, 0.6640288233757019, -1.2346168756484985, -0.04779166728258133, -1.2451326847076416, -0.9613367915153503, -1.3764008283615112, 0.2765912711620331, -0.5604491233825684, -1.1097817420959473, 0.5066201090812683, -1.0108176469802856, -0.5151920318603516, -1.1983963251113892, -0.9168599843978882, 1.4421888589859009, -0.18622276186943054, -1.1528664827346802, 0.6586501598358154, -0.4964248239994049, -0.09020132571458817, 0.1589718610048294, 1.3513884544372559, 1.0484634637832642, -1.4813687801361084, 1.046642541885376, 1.538170576095581, -1.0175244808197021, -0.5012051463127136, 0.2403106391429901, 1.1787309646606445, -1.2767549753189087, 1.6043494939804077, 1.2013884782791138, 0.317741721868515, 1.9035661220550537, -1.0923784971237183], [1.996508002281189, 0.08641110360622406, 1.3108819723129272, -1.3447552919387817, 1.3745760917663574, 0.3623647093772888, -1.312438726425171, -0.22787296772003174, 2.150524616241455, -0.7434682250022888, 0.14053210616111755, 0.9357025623321533, 1.518667221069336, 0.08026202023029327, 0.3971741497516632, 0.22638583183288574, 0.8528579473495483, -1.4706530570983887, -0.8476631045341492, -0.9210489988327026, 1.3920220136642456, -1.2840622663497925, -0.8988012075424194, 1.0164191722869873, -1.9585981369018555, -1.0465610027313232, -0.6108725070953369, 1.359671950340271, 1.4255973100662231, 2.3126883506774902, -2.993704319000244, -1.3874874114990234, 0.6143301129341125, -1.9221243858337402, 0.8063139915466309, -1.5399023294448853, 1.0185858011245728, -0.5303502082824707, -1.2559502124786377, -0.32695263624191284, 2.153719902038574, 0.8433046936988831, 2.090036392211914, -1.0708286762237549, -0.07000871002674103, 0.7943392395973206, 1.1748913526535034, 0.06659334897994995, 0.21436768770217896, -0.42988070845603943, -0.07080940157175064, -0.5812516212463379, 1.0262020826339722, 0.40308472514152527, 0.006723609287291765, 0.49892091751098633, 0.7401888966560364, -0.460604190826416, -0.371256023645401, 0.22593453526496887, -1.7546731233596802, -0.5850371718406677, -0.6892315149307251, 0.764668345451355], [-1.2901188135147095, 0.020473003387451172, 0.3233400285243988, -1.1375141143798828, 1.9081567525863647, -2.2590575218200684, -0.9953539371490479, 2.2312698364257812, 2.819115400314331, 0.2919973134994507, 0.9786405563354492, -0.5304405093193054, 0.08263450115919113, -0.2482275515794754, 1.043709397315979, 0.5629456043243408, -0.6697129607200623, 0.7435029745101929, -1.7635571956634521, 1.9518953561782837, -1.3774383068084717, -0.9983581304550171, 0.7627129554748535, 0.11856646090745926, 0.8499701619148254, -0.23036816716194153, 2.1337826251983643, -2.285409450531006, 1.0890907049179077, 2.695960760116577, 1.114061951637268, 0.317135751247406, 0.7814971804618835, -0.04022393748164177, 0.9380069375038147, -0.1841518133878708, 0.5552043914794922, 0.03728075325489044, 0.07296070456504822, -0.779215931892395, -0.41164374351501465, -1.555912733078003, -0.4568081796169281, -0.12271203845739365, 0.3523021638393402, -0.1099400594830513, -0.49601149559020996, -0.17680180072784424, -0.46651965379714966, 0.14701934158802032, 1.0340675115585327, -0.47204703092575073, 0.3694378137588501, -0.2135612964630127, -0.09220227599143982, 1.1975001096725464, -1.8690019845962524, -0.2060755342245102, -1.0548170804977417, -1.0123869180679321, -0.7715350985527039, -0.1284681260585785, 0.6082091927528381, -0.9689562916755676], [-1.2474989891052246, 0.35806700587272644, 1.465334177017212, 1.117425560951233, 1.1232800483703613, -0.07473570108413696, 0.8215402364730835, 0.8016750812530518, -1.8521602153778076, 0.481738418340683, -1.1002578735351562, 1.6999613046646118, 0.05665784701704979, -0.500060498714447, 0.5419171452522278, 0.2536711096763611, 0.9344347715377808, 0.827714741230011, -1.2280470132827759, -0.6827393174171448, -0.15774306654930115, 0.605900764465332, 1.5551056861877441, -0.45965126156806946, 0.4063553214073181, 1.031361699104309, 0.7200781106948853, -0.6824256777763367, 0.04375452920794487, 0.7539891004562378, -0.05832012742757797, -0.02319895476102829, -0.46999701857566833, -1.0856297016143799, -0.41241219639778137, -0.9388788938522339, -0.3876267671585083, -0.2471262365579605, 1.6873306035995483, 1.015700101852417, -0.13052795827388763, 0.5827577710151672, -2.350677013397217, 0.15519529581069946, 1.38551664352417, -1.7359105348587036, -1.158949375152588, -0.661357045173645, -0.5808553695678711, 0.4778587818145752, 0.8121609687805176, -1.0576281547546387, -1.0206950902938843, 0.7189271450042725, 1.030380368232727, 1.090140461921692, 0.40127941966056824, -0.4614313542842865, -0.13636818528175354, -1.11601984500885, -1.1178617477416992, -1.2615872621536255, 0.024051157757639885, 0.20130743086338043], [-0.663969099521637, 0.7995557188987732, 0.5369426012039185, 0.002145790494978428, -1.137333631515503, -0.8407456278800964, 0.30502432584762573, 0.29731619358062744, -0.9588124752044678, 0.9920663833618164, 0.7141096591949463, -0.19602802395820618, 1.7946892976760864, -0.7484628558158875, -0.035543292760849, -2.7489593029022217, -0.711776852607727, 0.6979305744171143, -0.8893195986747742, -1.3582650423049927, -0.8265739679336548, -0.486960232257843, 2.020353317260742, -0.5797953605651855, -0.3463629186153412, -0.7229843139648438, 1.8777228593826294, 0.8198115229606628, 0.5247331261634827, -0.2493753731250763, -1.3262466192245483, 1.765669345855713, -1.015419363975525, 0.5180742740631104, -0.21859139204025269, 1.4480770826339722, -0.011580993421375751, -0.3051691949367523, -0.581817090511322, -0.9165605306625366, 1.0266815423965454, -0.5303985476493835, 1.0130730867385864, 0.45001816749572754, -0.11824021488428116, -0.4650169312953949, -0.30790233612060547, 0.0350063256919384, -0.0997951477766037, -1.200459361076355, 1.5188223123550415, -0.1661575436592102, 0.6361232995986938, 0.8897782564163208, 1.2180192470550537, 0.5060712099075317, -0.9252456426620483, -0.5926746726036072, -0.6995473504066467, -1.3645051717758179, 0.46575552225112915, -0.2304844856262207, 2.1123509407043457, 0.08435209840536118], [-0.6300894618034363, 1.83566415309906, -0.1455450803041458, -0.14522069692611694, 0.3694973289966583, -0.4246174991130829, -2.447916030883789, -2.07246732711792, -0.13274817168712616, 0.5626733303070068, -0.27668559551239014, 0.7453561425209045, -0.6462580561637878, 1.0655146837234497, 0.5842932462692261, -1.1563912630081177, -0.04400604963302612, 0.6392338275909424, 1.7789403200149536, -1.3228927850723267, -0.5049335360527039, 0.19721537828445435, -3.112128973007202, -0.3441047966480255, -0.06272953748703003, 0.09515140950679779, 1.5885111093521118, 1.2461236715316772, 0.5186995267868042, 1.0833635330200195, -0.43495434522628784, 0.3169543743133545, 0.549990177154541, -1.0901373624801636, 1.150537133216858, -0.1669478565454483, -1.9095181226730347, 1.1403839588165283, -0.8215040564537048, -1.4262313842773438, 0.15052339434623718, -0.03435678780078888, 1.2676067352294922, -0.9185501933097839, 1.5013359785079956, 1.0269508361816406, 0.4038267135620117, 0.5027269721031189, 0.9591103196144104, -0.7972054481506348, 0.7311956882476807, 0.5648959875106812, -1.566750407218933, -1.4464983940124512, 0.048598311841487885, 0.17966262996196747, -0.901223361492157, 1.25188410282135, 0.2904493510723114, 0.9778818488121033, -0.07078235596418381, -0.25239846110343933, 0.9736960530281067, 0.08135128021240234], [1.4764312505722046, 0.2859351933002472, 0.38838160037994385, -0.5348414778709412, 1.068085789680481, 1.8057951927185059, -0.23737139999866486, 3.4231441020965576, 0.15032753348350525, -0.01199411042034626, 0.6430261731147766, -0.3185162842273712, 0.4022747278213501, 1.0370306968688965, 0.44768333435058594, 0.11370252072811127, -0.6689943671226501, 0.6243605017662048, -0.14957737922668457, -0.8231247067451477, 1.47602379322052, 1.9025695323944092, 0.25762122869491577, 1.0016531944274902, 1.61436128616333, 0.8322001695632935, 0.37267300486564636, 0.2315828502178192, 1.3931628465652466, -0.9982144236564636, -0.8310883641242981, 0.6114102602005005, -0.6684849262237549, -1.2028636932373047, 0.13625185191631317, -0.03335382416844368, -0.7195682525634766, 0.7182945013046265, -0.1416747123003006, -0.6427590847015381, -0.3733343780040741, -1.369177222251892, 0.13933251798152924, -0.1835576593875885, 0.7436398863792419, -0.3926229476928711, 1.1520506143569946, 0.48766523599624634, 0.41144976019859314, -0.5018351674079895, 0.21435262262821198, -0.7475334405899048, -0.3346719443798065, 1.0008296966552734, 0.7018142342567444, -0.4643677771091461, 1.9028269052505493, 1.112383246421814, 0.7738712430000305, 0.7617173194885254, 1.2022106647491455, -0.0912233218550682, 0.5479734539985657, 0.7806429862976074], [-0.9923449754714966, -1.4749407768249512, 0.3072822093963623, -0.6114837527275085, 0.2798720896244049, -1.269793152809143, 0.6153466701507568, 1.5459872484207153, 0.12027372419834137, 1.4340656995773315, -0.07841476052999496, -0.42092207074165344, -0.49965983629226685, -2.070422410964966, -0.005637360271066427, 0.14242100715637207, 0.36555200815200806, -0.175174742937088, -0.099821537733078, -0.30373501777648926, -1.86897611618042, -0.841509997844696, 0.30759015679359436, -1.2255609035491943, 0.803866446018219, 1.8362350463867188, 1.8545925617218018, -0.14204616844654083, -0.4241408407688141, 1.2836079597473145, -0.5961849689483643, -0.6248025894165039, 1.4060499668121338, -1.238849401473999, -0.8854982852935791, 0.8212612867355347, -0.25308817625045776, -0.2350536435842514, 0.8225547671318054, 0.9437452554702759, -0.32607635855674744, 0.19290724396705627, 0.8223180770874023, 1.9302417039871216, 0.09638053178787231, -0.5904714465141296, -0.018125373870134354, 0.6289617419242859, 0.5269557237625122, -1.601881742477417, -0.22195270657539368, -0.8140097856521606, 1.1399801969528198, 0.39076387882232666, -0.37341493368148804, 2.134692668914795, -1.2475998401641846, -0.3189769983291626, 0.7740548849105835, 0.6381967067718506, -0.5189910531044006, 0.4593605101108551, 0.773316502571106, 0.9453734755516052], [-0.8775589466094971, -0.9187382459640503, -0.5094241499900818, -0.33576950430870056, 0.3639453947544098, -1.616464614868164, 2.778794288635254, 1.3525397777557373, 0.4750608205795288, 0.048741765320301056, 1.6309183835983276, 0.0569160170853138, 0.4232271909713745, -1.101260781288147, 0.20932158827781677, -1.3282233476638794, -0.3914427161216736, 0.742676317691803, -0.9810468554496765, -0.19538816809654236, -1.615723729133606, -0.8057589530944824, -0.5024837851524353, 0.3315887451171875, 1.1073817014694214, 1.2199519872665405, -0.44235193729400635, -0.9519596695899963, -0.37480100989341736, -0.6210317611694336, -0.3663500249385834, -1.279200553894043, -0.2128491997718811, 0.09171521663665771, 0.0256052166223526, -0.6312025189399719, -0.37164098024368286, 0.3151082992553711, 0.2093677967786789, 0.2778266370296478, 0.1395806223154068, -0.2803475260734558, 0.751820981502533, -0.970319390296936, 1.948349952697754, -0.7532632946968079, 0.35396185517311096, 1.616392970085144, 0.7906880974769592, 0.6041668057441711, -0.4568193852901459, 1.370819091796875, -0.3044435977935791, 0.34571707248687744, -0.398257315158844, -0.17073224484920502, -0.73673415184021, -1.3000491857528687, 1.1029006242752075, -0.9088062047958374, -0.015145082026720047, -0.2773882746696472, -0.5041791200637817, -0.6182844638824463], [0.1172352060675621, 0.05156558007001877, -2.3826141357421875, -0.9869762063026428, -0.23864729702472687, 0.7493827939033508, -0.12188761681318283, 1.1214648485183716, -0.13906678557395935, 0.023066259920597076, -0.273506760597229, 0.3027508854866028, -2.6428332328796387, 0.14131906628608704, 0.2536396086215973, -0.37358400225639343, -0.6528441309928894, 0.7266069054603577, -1.1340376138687134, 0.3053279221057892, 0.5146104693412781, -1.7715649604797363, -1.6964929103851318, -0.39637765288352966, 0.7607346773147583, -0.3900768458843231, -0.7105863094329834, 0.3825356364250183, 0.18612121045589447, -0.6396470069885254, 0.17906148731708527, 1.1366349458694458, -0.2811450660228729, 1.3919709920883179, 0.4761494994163513, 1.6217516660690308, 0.5991334915161133, 0.6980730295181274, -0.5535948872566223, 2.360935926437378, -1.5122928619384766, 0.722973108291626, 0.061902716755867004, -1.3476834297180176, -0.809354305267334, -0.5664619207382202, 0.6531075835227966, -0.15695396065711975, -0.2110522985458374, -3.272679090499878, -1.2971993684768677, -0.7718679904937744, -0.42996111512184143, -1.6068991422653198, 1.8468666076660156, -0.3271321654319763, 0.8998811841011047, -1.6724525690078735, -0.9032995700836182, 0.5209499597549438, -0.328082412481308, -0.5634551644325256, 1.4227224588394165, 0.013691144064068794], [-0.8087970614433289, -1.1125609874725342, 0.3819234073162079, 0.5581685304641724, 1.349461317062378, 0.25133469700813293, 0.15148267149925232, 0.475730299949646, 2.309997320175171, 0.27040523290634155, 1.3635239601135254, -0.4248751401901245, -0.7073823809623718, -0.9627370238304138, 0.03841901570558548, -0.42465049028396606, 0.8620533347129822, 0.7250973582267761, 0.07366704195737839, -0.21472136676311493, -1.1742273569107056, -0.008106537163257599, 0.511496901512146, 1.8993550539016724, 0.7472545504570007, -1.030758023262024, 1.6758275032043457, 0.7487645149230957, -0.018341418355703354, -1.189374566078186, 0.2977895140647888, -0.9254221320152283, 0.03847305849194527, -1.5111382007598877, -0.5790376663208008, 0.16073814034461975, 0.6924355030059814, 0.9755429029464722, 1.1507842540740967, -0.23907433450222015, 0.27874448895454407, 1.1223704814910889, -0.7548146843910217, 0.15293703973293304, 0.12750020623207092, -0.1261242926120758, 2.0517654418945312, -0.9991444945335388, 0.28531068563461304, -0.27705076336860657, -0.5008499622344971, 0.5414227247238159, -1.293707013130188, -0.7479076981544495, 0.20398208498954773, 0.7368305921554565, 0.008434930816292763, -0.21143648028373718, -2.3218743801116943, -1.1152150630950928, -0.8593639731407166, -1.9247325658798218, 0.990803062915802, -0.6473472118377686], [0.07181411981582642, -0.6519342064857483, 0.3319482207298279, 2.087502956390381, -1.4470939636230469, 0.04311120882630348, 0.5056166648864746, -1.1301127672195435, -1.351857304573059, 0.886946439743042, 1.3307636976242065, 1.3306653499603271, -1.8695605993270874, 2.5331244468688965, -0.600114643573761, -0.12313149124383926, 2.5101735591888428, -0.7815437316894531, 2.5205304622650146, 0.13628363609313965, -1.5777077674865723, 2.0506064891815186, -0.4333832263946533, -1.1588351726531982, 0.08944623172283173, 1.341809868812561, 0.5365514159202576, 0.7201502919197083, -1.1969757080078125, -0.5206983089447021, -0.3670833706855774, 1.5721182823181152, 0.26364707946777344, -0.2856065034866333, -1.5998477935791016, -1.497666597366333, 0.09559670090675354, 1.049782395362854, 0.7606340050697327, 0.5206311345100403, -0.8651037812232971, 0.5896475315093994, 1.1693049669265747, -1.7779414653778076, 2.140326976776123, 0.20744504034519196, -0.8591791987419128, 1.755409836769104, -1.8986343145370483, -2.9495203495025635, -0.8988288640975952, 0.3390250504016876, -0.1168571263551712, 0.5329219698905945, -0.7187793850898743, 0.9328434467315674, 1.1550379991531372, 0.2762061059474945, -0.25969749689102173, -0.2008938491344452, -1.0344390869140625, 2.5171873569488525, -0.26769015192985535, 0.9134128093719482], [-1.4844456911087036, -0.35610395669937134, -0.10965461283922195, -0.18221253156661987, 0.20705267786979675, -0.34546685218811035, -0.04063970223069191, -0.022343313321471214, -0.6170628666877747, 1.4321919679641724, 0.7252956628799438, -0.49257904291152954, 0.6185025572776794, 0.8271253705024719, -0.17988547682762146, -0.8419985771179199, -0.7743129730224609, 1.9901957511901855, -1.0843427181243896, -0.34124523401260376, -0.036197420209646225, -0.7212685942649841, 0.10790996253490448, 0.04163016006350517, 0.49497678875923157, -0.4084993302822113, 0.9521082639694214, 0.27177244424819946, 1.7383310794830322, 1.1864389181137085, 0.09492229670286179, -1.3250354528427124, -0.16960306465625763, -1.095820665359497, 1.5498982667922974, -0.12045329064130783, -0.4742981493473053, -1.1015018224716187, -0.8328718543052673, 1.1351121664047241, -0.47953930497169495, 1.3738147020339966, -0.08282195031642914, -0.08579310029745102, -2.101907730102539, -0.02063450589776039, -1.0872405767440796, 0.7004830241203308, 0.18480068445205688, -1.0386170148849487, 1.1733566522598267, -0.8927441239356995, 0.43378135561943054, -0.2886607050895691, -0.2534105181694031, -1.1143426895141602, 1.4015978574752808, 0.48302140831947327, -1.4354702234268188, -0.9698762893676758, 0.5584094524383545, -0.3340882956981659, 0.11554157733917236, 0.34803882241249084]]], [[[-0.1381947249174118, -1.010888695716858, 1.6456536054611206, 1.043554425239563, 0.641879677772522, -0.47027260065078735, 0.3403353989124298, -1.637318730354309, 0.31059178709983826, -1.0324575901031494, 0.80331951379776, -0.9692904353141785, -1.2482426166534424, -0.9580772519111633, -0.349570631980896, -2.395490884780884, 1.02120840549469, 0.1742871254682541, 2.193549394607544, -0.40297234058380127, -1.4282649755477905, 0.9330501556396484, -0.4895865321159363, -0.1309286504983902, -0.047895535826683044, 0.4803883135318756, 0.00036197411827743053, 0.4494345784187317, -0.8527477979660034, 0.09871921688318253, -0.5783620476722717, 0.4067985713481903, 0.35303765535354614, 1.1843852996826172, -1.3189589977264404, 0.4158176779747009, -0.4868813455104828, 0.35888752341270447, -1.0400264263153076, 0.721124529838562, 0.07362053543329239, -0.009954816661775112, 1.8179506063461304, 0.8250675201416016, -0.11009766906499863, 0.9191710948944092, -0.49459218978881836, 0.2183225005865097, 0.46514394879341125, 2.0810329914093018, 2.092360019683838, 0.42165613174438477, 0.5897051692008972, -1.5766115188598633, -0.3534921705722809, -1.1650294065475464, 1.7173633575439453, 0.046504098922014236, -0.9440734386444092, 0.3013751804828644, 1.4529871940612793, 0.17433778941631317, -0.25926175713539124, 1.1157087087631226], [0.3603856861591339, -0.19759204983711243, -1.4541035890579224, -0.15644890069961548, -0.14616918563842773, 0.8535190224647522, -0.18719595670700073, -0.1969655454158783, 0.3844602108001709, 0.8258597254753113, 2.0108067989349365, -1.2293431758880615, -0.7758119106292725, 1.8345099687576294, -0.07525230944156647, -0.8250781893730164, 0.35499870777130127, 1.5823025703430176, 0.33051419258117676, 0.15427318215370178, 0.3558332026004791, -0.4041898548603058, -1.0460964441299438, 2.1138546466827393, -0.5906659960746765, 0.03172648698091507, 1.42988920211792, -1.124498724937439, -0.7889022827148438, 0.960867166519165, -0.054666101932525635, 0.32112205028533936, 0.2336588054895401, 2.23572039604187, -0.6754054427146912, -0.8338848352432251, 0.7780001163482666, 1.135817050933838, -0.28879469633102417, -0.3203764855861664, -2.0567471981048584, -0.9597899317741394, -0.13978539407253265, 0.006344988010823727, -0.1446458250284195, 0.17440220713615417, 0.3829522728919983, 1.7985862493515015, -0.3145253360271454, 1.0374982357025146, -1.0583378076553345, -0.15215668082237244, -0.4811697006225586, -0.6461989283561707, -0.12981659173965454, 0.7955266833305359, -0.4756702780723572, -1.8531233072280884, 1.0557363033294678, 1.974324107170105, 0.5996620059013367, 0.02336476556956768, -0.1830005645751953, 0.5410553812980652], [2.1879780292510986, 0.6724725365638733, -0.6934752464294434, 0.7672923803329468, -0.8942654132843018, 1.2651159763336182, -0.03594416379928589, 0.7445619702339172, -0.20458011329174042, -2.227382183074951, 0.3187079429626465, -0.5541889071464539, 0.14796197414398193, 1.306410789489746, -1.322993516921997, -0.7481926679611206, -1.166988730430603, -0.05214514210820198, 0.3211866021156311, 0.02934529259800911, 0.433289110660553, 0.884604275226593, 1.2009330987930298, -2.2927305698394775, -1.4063972234725952, 0.01705646701157093, -0.8951613306999207, 0.7812317609786987, 1.2832293510437012, -0.9088362455368042, -0.020795177668333054, -0.19019284844398499, 1.0426193475723267, -0.5563148260116577, -0.15895888209342957, 0.7616363167762756, -0.09478789567947388, -0.7054675221443176, 1.5714842081069946, -0.9216277599334717, -0.7346015572547913, 2.0279526710510254, -0.027256395667791367, 0.9563698172569275, -0.994199812412262, -2.1476123332977295, 0.9094680547714233, -0.11861547827720642, -1.1559230089187622, 0.21687103807926178, -0.17896172404289246, -0.04178827255964279, 0.8629907369613647, -0.7816668748855591, 0.8083857893943787, -0.43306148052215576, 0.7003906965255737, 1.1607213020324707, 1.6878571510314941, 1.198935866355896, -1.2071900367736816, -1.2366849184036255, 1.1824573278427124, -0.6931714415550232], [-0.5483407974243164, -0.6146999001502991, 2.519216537475586, -1.6103076934814453, 0.004777722992002964, 0.0860762670636177, 0.5288235545158386, -0.38092637062072754, -0.2510833144187927, 0.25671687722206116, 0.22421661019325256, -0.2630936801433563, 0.13736124336719513, 0.47631657123565674, -1.1629101037979126, -0.9732142686843872, 1.4627606868743896, 0.18768750131130219, 1.3252322673797607, -0.754182755947113, 0.8083850145339966, -1.015851378440857, 0.8240447044372559, 0.1722269505262375, 1.1671926975250244, -0.9007735848426819, 0.9096366763114929, -0.5721641182899475, -0.8452796339988708, 0.571258544921875, -1.134080410003662, 0.4919615089893341, -0.12199939787387848, 0.506162703037262, 0.14834406971931458, 0.10151249915361404, -0.2596858739852905, -0.5617241263389587, -0.9058637022972107, 0.6702986359596252, -0.255083292722702, 1.753920555114746, 1.0937068462371826, -0.4542454183101654, 0.49546751379966736, 1.2256767749786377, -0.7961490750312805, -1.893797755241394, 0.9757081270217896, -0.7272136807441711, -1.6787720918655396, 0.1761254072189331, -0.3869483172893524, 0.6995066404342651, 1.4556081295013428, 1.7510865926742554, 0.6957346200942993, 0.7486576437950134, -1.0942045450210571, 1.345004916191101, 0.7510249018669128, -0.7552568316459656, -0.2929602861404419, -1.036940097808838], [-0.10376632213592529, -0.42488035559654236, 0.06119374930858612, -1.8599605560302734, 1.1867321729660034, -0.051807042211294174, -0.5871240496635437, -0.13920146226882935, -1.007132649421692, 0.5388684272766113, -0.13234686851501465, -0.751606285572052, 0.16687604784965515, -0.6438098549842834, -0.08488720655441284, -0.5632094144821167, 1.474257469177246, 0.5490497350692749, 0.3353990614414215, 0.5300071239471436, 0.8431238532066345, 1.7612402439117432, 0.07438197731971741, 0.6598042845726013, 0.5806466341018677, 0.8949946165084839, 0.634346067905426, -0.6518761515617371, 0.26683729887008667, -0.6775082945823669, -0.6037220358848572, -0.24994339048862457, -0.8069313168525696, 1.3725370168685913, -0.42962607741355896, 0.9745369553565979, 0.4316500127315521, -0.4374271035194397, 0.9488667249679565, -3.0322108268737793, -0.9825255870819092, 1.8125609159469604, 0.11017833650112152, -0.4304686486721039, 0.7969231009483337, -1.2141494750976562, -1.7004917860031128, -0.6877467632293701, -1.4547467231750488, -0.5658293962478638, -0.3835189640522003, -0.22086840867996216, -0.42439648509025574, -1.6292603015899658, -0.2520989775657654, -1.982629418373108, -0.6938722133636475, 1.7100268602371216, 0.4324895739555359, -1.1039224863052368, 0.19299177825450897, -0.08561372011899948, -0.5560290813446045, -1.3751596212387085], [0.15151169896125793, 0.2434602975845337, 0.785773754119873, 0.4561866819858551, 1.9754674434661865, 0.07401866465806961, -3.5269598960876465, -1.221888780593872, 0.2700630724430084, -1.1179602146148682, 1.2079943418502808, -1.7790645360946655, -0.6838284134864807, -0.6564601063728333, -0.7786406874656677, 1.3596372604370117, -0.6778258085250854, 0.4664056897163391, -0.7531455159187317, 1.309615135192871, 0.8819910287857056, 2.0653650760650635, -1.0113542079925537, -0.07778719067573547, 0.551493763923645, -0.2250053584575653, -0.8699713945388794, -1.7411236763000488, 0.2454504668712616, 0.2149377167224884, -0.06298650056123734, -0.017357345670461655, -0.3924051821231842, -0.4594787061214447, -2.8271641731262207, 0.601815938949585, -0.058316320180892944, 0.22621706128120422, -1.1280720233917236, -0.906105101108551, -0.5881222486495972, -0.07925226539373398, -0.5883989930152893, 0.5542637705802917, 0.29943564534187317, 0.37188076972961426, 0.4066994786262512, -0.03139590844511986, -0.7362465262413025, -0.8253023624420166, -1.644286036491394, -2.0498039722442627, -0.7428134083747864, 1.0202833414077759, -0.4502522945404053, -0.3559405207633972, -0.034260135143995285, -0.4308425486087799, -1.4185696840286255, -1.1634151935577393, 1.5580384731292725, -0.35859158635139465, 1.142473816871643, 0.8004852533340454], [-0.5174044370651245, 1.5506336688995361, 0.5110357999801636, -1.9554469585418701, -0.6696092486381531, 0.4199124574661255, -1.7386000156402588, 0.10796462744474411, 0.8928626179695129, -0.28401118516921997, 0.2130529135465622, -0.16590455174446106, 0.4673893451690674, 0.5103428363800049, -0.3328480124473572, -2.3695619106292725, -0.8476943969726562, -0.4949849247932434, 1.1422886848449707, -0.04283105581998825, -2.1300172805786133, 0.37622493505477905, -0.846796452999115, -0.3750384449958801, 0.6296527981758118, 0.7113891243934631, 2.7826921939849854, 1.1983537673950195, 0.0036916618701070547, 0.8514811992645264, 0.31044965982437134, -1.1405121088027954, -0.23484428226947784, 1.0877807140350342, 0.9097931981086731, 0.5477544665336609, 1.5108529329299927, 0.5317018032073975, 1.0679398775100708, -0.46985703706741333, -0.8187240958213806, 0.3754863739013672, 0.9476084113121033, 0.5764954686164856, 0.620622456073761, 1.62830650806427, 0.1036040261387825, 0.793591320514679, -0.2167225331068039, -0.046653635799884796, 0.8390509486198425, 2.0027880668640137, -0.26363739371299744, -1.0306788682937622, 1.1329033374786377, 1.548959732055664, 0.37427088618278503, -0.24942079186439514, 0.20713143050670624, -0.14288420975208282, -1.4485104084014893, 0.24054470658302307, 1.0481668710708618, -0.526020348072052], [1.4934685230255127, -1.918393611907959, 1.0062718391418457, -1.5745880603790283, -0.47217655181884766, -0.9591631293296814, -0.9813115000724792, 2.2057242393493652, 0.00361064774915576, 1.6808359622955322, 2.0980870723724365, 0.5476152300834656, -0.3094250559806824, -0.3183049261569977, -0.9133190512657166, 0.3735787868499756, 1.2430529594421387, -0.7793294191360474, -0.8237756490707397, -0.04408667981624603, 1.6917836666107178, -0.22032348811626434, 1.6930510997772217, -1.2110638618469238, 0.5003395676612854, 1.6070125102996826, -0.8057965040206909, -0.5293818116188049, 1.0145018100738525, -0.1714615374803543, 0.5417417883872986, 0.6597731709480286, 1.2126590013504028, 1.8490939140319824, -0.8935555219650269, 0.11363638192415237, 1.1567258834838867, 0.8070096373558044, 0.7738924622535706, -0.6375279426574707, 0.6961275339126587, -0.8286320567131042, 1.5739965438842773, -0.26958101987838745, -2.7348530292510986, -2.843966484069824, 0.7632014751434326, 1.8495872020721436, 0.45514237880706787, 0.6343014240264893, 0.2185051292181015, -0.00021515499975066632, -0.31086137890815735, -1.653571367263794, -0.035153016448020935, 0.5835023522377014, -0.42455336451530457, 0.3681681156158447, 0.8050702810287476, -1.6696693897247314, 0.23026694357395172, -0.254885196685791, 1.1802427768707275, 0.6063275337219238], [0.45702821016311646, -1.803636908531189, 0.844356119632721, -0.843498945236206, -0.8400782942771912, -0.36785247921943665, 0.5212110280990601, -0.4955922067165375, 2.060671329498291, 1.801753044128418, -0.5531667470932007, 0.16140824556350708, -0.8455464243888855, -1.369185447692871, -0.992654025554657, 0.8641502261161804, -0.1252424269914627, -0.7145529985427856, -0.6069270372390747, -1.2103816270828247, -0.19077341258525848, 0.47987228631973267, 1.6234309673309326, 0.49736616015434265, -0.3566955029964447, -0.7719172239303589, 2.7637827396392822, -0.3646998405456543, -0.0202327910810709, -1.7627086639404297, -0.30997371673583984, -0.4344753324985504, 0.1900818645954132, 1.2685935497283936, 0.7055661082267761, -0.8991524577140808, -1.5819804668426514, -0.17899231612682343, 0.492775559425354, -0.7202668190002441, -0.42802736163139343, 0.3514484763145447, -1.0021454095840454, 0.6346113681793213, -0.7759697437286377, 1.8846770524978638, -0.02434002235531807, -0.5541463494300842, -0.29208728671073914, -1.088049292564392, 0.24675355851650238, -1.308998703956604, 0.8512901663780212, -0.12723548710346222, -1.3989498615264893, 0.598767101764679, 0.3616858422756195, -1.3116480112075806, 0.44790077209472656, -0.22203952074050903, 0.289366751909256, -1.4789578914642334, 1.1919533014297485, -0.6255374550819397], [0.333749920129776, 1.4728518724441528, -0.10408619791269302, 0.6943622827529907, 1.2830051183700562, -0.9817785024642944, -0.6732786893844604, 0.2888522446155548, 0.9233606457710266, -0.5647050738334656, 1.4293580055236816, 0.9955511689186096, -0.03587799519300461, -0.5648581385612488, 1.8201402425765991, -0.6586771607398987, 0.7385415434837341, -2.3101487159729004, 0.11645790189504623, 0.16511230170726776, -0.20040272176265717, 0.9843433499336243, 0.6201378107070923, -1.2558435201644897, 1.185249924659729, -0.013208632357418537, 1.572298526763916, -0.2571812868118286, -0.1687449812889099, 0.5325621366500854, -0.8824224472045898, 0.4450269043445587, -0.18777428567409515, -1.0126686096191406, 0.9155533909797668, 0.6662111282348633, -1.1765234470367432, 0.8700821399688721, 1.299789309501648, 0.6978880167007446, -0.8059459328651428, -0.33108144998550415, -0.45614880323410034, 0.7944746613502502, -1.5574874877929688, 0.3269787132740021, 0.4845273196697235, -2.010056734085083, 1.239193081855774, 0.4800659418106079, -1.7669016122817993, -0.4182935655117035, 0.8485519289970398, -0.9344991445541382, -1.9082621335983276, -0.9052777886390686, -0.5795618295669556, 0.27629029750823975, 0.6038072109222412, -0.8654646277427673, -2.082045316696167, -0.579657793045044, -0.7515884041786194, -0.05290907993912697], [1.2314743995666504, -1.1729902029037476, 1.6400971412658691, 0.3849848806858063, 1.0456244945526123, -2.3417346477508545, -0.15211234986782074, 0.6733006238937378, -0.531130313873291, -0.12791390717029572, -0.4466612637042999, -0.3967263698577881, -0.52825528383255, 0.9102756381034851, -0.6954976320266724, -0.5215584635734558, 1.2963110208511353, -0.05588262528181076, -0.4482376277446747, 2.5020642280578613, -2.3961124420166016, 0.6187834143638611, 0.1229967400431633, 2.455721616744995, 0.3803538978099823, -0.8127440214157104, -0.18434345722198486, -0.9744108319282532, 1.763518214225769, 0.15738245844841003, 0.228117898106575, -0.23588529229164124, 0.646166980266571, -0.9622747898101807, 0.11936915665864944, -0.4201800525188446, 0.38179436326026917, -0.15483535826206207, 0.32987263798713684, 1.292147159576416, 0.06334470957517624, 0.6779922842979431, -0.38461264967918396, -0.649545431137085, -0.18720197677612305, -0.38024649024009705, 0.16952306032180786, -0.2974901497364044, -0.3701842427253723, -1.0593035221099854, 0.7046212553977966, -0.6509758234024048, -0.41367393732070923, 1.1095768213272095, 1.998611569404602, 1.4478235244750977, 1.0159547328948975, 1.5395406484603882, -0.5828961730003357, 1.296472430229187, 0.7756607532501221, 0.07048378139734268, 0.22201792895793915, 0.5348454713821411], [-1.1599208116531372, 0.8871437311172485, 0.45420581102371216, 0.10602313280105591, -1.2276360988616943, 0.651050865650177, -0.7307875156402588, 1.3873615264892578, -0.30329668521881104, 0.18324504792690277, -2.2757821083068848, 0.677388072013855, -0.10720115900039673, 1.0162253379821777, 1.3063050508499146, -0.9446171522140503, -1.3065649271011353, -0.640247106552124, 0.774061381816864, 0.046341340988874435, 0.4913913309574127, -0.45208755135536194, -0.46349871158599854, -0.8157921433448792, 1.3432273864746094, 1.1058207750320435, 0.6596536636352539, 2.212134599685669, 1.654234766960144, -0.6803878545761108, 1.1553601026535034, 0.7008917331695557, 0.9626690149307251, -0.8523895144462585, 0.6998773813247681, 0.10575052350759506, 1.5863280296325684, 0.3113170564174652, 0.4531663656234741, 1.2708722352981567, -0.43231329321861267, 1.4781829118728638, 1.9055699110031128, 0.395920991897583, -0.11045685410499573, -0.1144433245062828, -0.22452689707279205, 0.5398847460746765, -0.33241358399391174, 2.003830671310425, 0.46424537897109985, 0.8324267864227295, -0.014392365701496601, 0.48725706338882446, 0.6234115362167358, -0.11800512671470642, -0.28849557042121887, -1.438502311706543, -0.994087278842926, 0.12897397577762604, 1.9276483058929443, 0.12535423040390015, -0.41686397790908813, 0.7098380923271179], [0.6318171620368958, 0.23555588722229004, 0.25616979598999023, -1.7417831420898438, 1.5843356847763062, -1.265227198600769, -1.2636888027191162, -1.0653390884399414, -1.0149686336517334, -0.6428592801094055, -0.32534343004226685, 1.137855887413025, -1.1525840759277344, -0.13503585755825043, -0.22047515213489532, -0.20154190063476562, 0.09199944138526917, 1.9562021493911743, -0.2756762206554413, 0.22250770032405853, 0.25261273980140686, 1.4145053625106812, 1.28972327709198, 0.2497473657131195, 0.5443851947784424, 0.41735029220581055, -0.35749781131744385, -0.0014471383765339851, 1.0118188858032227, -1.9551002979278564, -0.44758856296539307, 1.158389925956726, 0.1346183866262436, 0.1719084084033966, 0.2388676404953003, 1.4240175485610962, 0.9205069541931152, -0.8550165891647339, 1.190624475479126, -0.32849037647247314, -0.5276917219161987, -0.24827787280082703, -1.29964280128479, 0.15287061035633087, -0.26303979754447937, -1.6222939491271973, 0.42036768794059753, 1.450477123260498, -1.0999783277511597, -1.1207869052886963, -2.178100824356079, 1.2485156059265137, -0.011144801042973995, -0.8152625560760498, 1.0820852518081665, 0.27824175357818604, 2.645566940307617, -0.1866065412759781, 0.7544766068458557, -0.1388787180185318, 1.1766908168792725, -0.8280033469200134, 1.372958779335022, -0.20308496057987213], [0.1693515032529831, 0.23036104440689087, 0.0753839984536171, -0.3379809260368347, -0.5395922660827637, -0.27986228466033936, 0.4392601251602173, -0.03681855648756027, 0.8722963929176331, 0.041257426142692566, 1.1011477708816528, -0.4502023160457611, 1.1437747478485107, -2.206181764602661, -1.405090570449829, 0.1012272760272026, 0.8671480417251587, 1.4724653959274292, -0.13858802616596222, 1.2291747331619263, -0.02470695972442627, -1.1390926837921143, 1.1387284994125366, 0.4866555631160736, -0.18451784551143646, 0.4240858852863312, 0.38679206371307373, -0.0065453690476715565, -0.027642622590065002, -0.6204506158828735, 0.583794116973877, -1.0846387147903442, -0.6703898906707764, 1.4941751956939697, -0.21019414067268372, 1.0962533950805664, -1.2651764154434204, -1.8944867849349976, 0.5561314821243286, -0.15348899364471436, -0.20372578501701355, -1.200441598892212, 0.5052489042282104, -1.8935153484344482, 0.09751754999160767, 1.4589464664459229, -0.32943102717399597, 0.6081149578094482, -2.1343834400177, -0.9765879511833191, -0.3552951216697693, 0.6440419554710388, -0.6363621950149536, 0.23533189296722412, 0.10079850256443024, -0.6904414296150208, 0.3831084370613098, -1.2361682653427124, 0.477484792470932, 1.2753100395202637, -0.4408130347728729, -0.6975442171096802, -0.41048210859298706, 0.07097875326871872], [0.12036346644163132, 0.027857020497322083, -0.926244854927063, 0.25810620188713074, 1.1367337703704834, -0.3001821041107178, 1.5943522453308105, 0.9836565852165222, 1.00142502784729, 0.4911838471889496, -0.14149878919124603, 1.2003779411315918, -0.15667827427387238, -0.45112746953964233, 1.3256815671920776, 0.6878237128257751, -1.8332717418670654, 1.061937928199768, 0.30619025230407715, -0.23204803466796875, 0.06973563134670258, 0.11592838913202286, -0.8893889784812927, 0.978636622428894, 1.803151249885559, 1.3864381313323975, 0.5151457190513611, -1.1640093326568604, -0.5563866496086121, -1.677516222000122, -0.21263344585895538, 0.035552285611629486, 0.24527597427368164, -1.3859037160873413, -0.320792555809021, -1.5833600759506226, -1.3927323818206787, -0.12313906103372574, -0.6137268543243408, 1.303622841835022, 1.3447312116622925, 0.6052956581115723, 1.1216405630111694, 0.1237095296382904, -2.192207098007202, -0.8572380542755127, 0.11746706068515778, 1.9416508674621582, -0.6600398421287537, -1.1740590333938599, -2.6152873039245605, -0.30711305141448975, -0.18655288219451904, -0.6122815608978271, 0.2057551145553589, 1.3651012182235718, -0.7227754592895508, -1.4828625917434692, 1.191133975982666, -1.1081688404083252, 0.1881529539823532, 1.6552884578704834, 0.047489456832408905, 1.2720025777816772], [0.8116713762283325, -2.562244176864624, -2.472961187362671, -2.133685350418091, -1.6024397611618042, -0.7117371559143066, 2.35969614982605, 0.9085426330566406, 0.04270664229989052, -1.2238283157348633, 2.188469648361206, -0.7494634985923767, 0.3501553535461426, -1.1231659650802612, 0.5902402997016907, 1.2873951196670532, 0.7526429295539856, -1.0776621103286743, -0.7210087776184082, 0.9142172336578369, -0.8201493620872498, 0.47661644220352173, -1.0137940645217896, 0.1679125428199768, -2.2980422973632812, 0.6625785827636719, 0.11341046541929245, -0.8602260947227478, 0.3764364421367645, -0.6019319891929626, -0.6090534329414368, -1.9560526609420776, 0.6975065469741821, -1.260946273803711, 1.7201279401779175, -0.22950786352157593, -0.6532708406448364, 0.07754950225353241, -2.085042715072632, -1.830549955368042, -0.024768801406025887, -0.5373451709747314, -1.0525233745574951, 0.6692802906036377, 0.38364726305007935, -1.2398136854171753, -2.253085136413574, 0.9682740569114685, -0.028563646599650383, 0.9532544612884521, -2.098395586013794, 1.6403875350952148, 0.05898125469684601, -0.7054075598716736, 1.1408369541168213, 0.21305902302265167, -1.317258596420288, 2.211751937866211, -2.070161819458008, -0.21281060576438904, 1.093808889389038, 1.2900470495224, 0.4188418388366699, -1.445914387702942]], [[-1.3032097816467285, -1.4085551500320435, 1.257759690284729, -0.3903457522392273, -2.0043420791625977, 0.6879453659057617, -0.8676369786262512, -0.4051544666290283, 0.08926921337842941, -0.8459975123405457, -1.8993531465530396, -0.9174767136573792, -0.10938658565282822, -1.3445030450820923, -1.2883301973342896, 1.5708588361740112, 0.28874149918556213, -0.22959227859973907, -0.7086868286132812, 0.39481720328330994, 0.26903894543647766, -1.6405611038208008, -1.0465071201324463, -0.02242831327021122, 0.5132469534873962, 1.0133219957351685, 1.1499049663543701, -0.257793128490448, -0.343418687582016, 0.46145230531692505, -1.4985921382904053, -1.092057466506958, 0.9258062839508057, -1.5086678266525269, 0.26615577936172485, 0.2800973057746887, -1.6301958560943604, 0.04852263256907463, 0.5525507926940918, 1.4505118131637573, 1.5981765985488892, 0.0864635780453682, -0.22426117956638336, -0.006411214359104633, -0.9308244585990906, -0.8712772727012634, -0.9833381772041321, 2.0533626079559326, -0.0013330749934539199, 0.019905518740415573, 1.6542667150497437, -0.8342505097389221, 0.6940916776657104, 0.9622942805290222, 0.16763071715831757, -0.008901608176529408, 0.9018924236297607, 1.7586411237716675, -0.34705209732055664, 2.8943638801574707, 1.6327780485153198, -1.6984455585479736, -0.025865139439702034, -0.5131883025169373], [1.322936773300171, 2.2946534156799316, 0.7409218549728394, -0.1232469230890274, -0.6332864761352539, 1.141118049621582, -0.8617064952850342, 2.272268056869507, 1.1405768394470215, 2.0610954761505127, 0.2042611986398697, 1.9521714448928833, -0.26406678557395935, 0.39606937766075134, 0.7642431855201721, -0.32181671261787415, -0.2428097277879715, 1.7416540384292603, -1.0879478454589844, 2.13153338432312, -0.3908078968524933, -0.6114645004272461, -1.4880280494689941, -1.4631530046463013, -1.0565649271011353, 0.043497323989868164, 0.47842511534690857, 0.567094087600708, 0.10118638724088669, 0.017955457791686058, 1.528372049331665, 0.13258656859397888, 0.15945599973201752, -0.11496848613023758, -0.6673130393028259, -0.286099374294281, -0.39111945033073425, -0.13604110479354858, 3.5178210735321045, 1.3668889999389648, -1.01339590549469, -1.4137153625488281, -0.5022480487823486, 0.07745496183633804, 0.38862141966819763, 0.1477193534374237, -1.6568145751953125, -1.7813773155212402, -0.5138444304466248, -0.7832794785499573, -2.285306930541992, -1.442802906036377, 0.1341405212879181, -1.2943392992019653, -0.5389161109924316, 0.33881667256355286, 1.2698975801467896, -0.9180983304977417, 0.0445450097322464, -0.09201318770647049, -0.758849024772644, -0.7403687238693237, 1.3833351135253906, -0.6452460289001465], [-0.03079674020409584, 0.13655142486095428, 1.6838027238845825, 0.08240833133459091, 1.7578758001327515, 1.1219369173049927, -0.6138486266136169, 0.7913243174552917, -0.8650104403495789, -0.11599523574113846, -0.39066535234451294, 1.4669184684753418, -0.8504008650779724, 0.9801990389823914, 0.22092624008655548, 0.8349664211273193, -2.3756494522094727, -0.19557863473892212, -0.17326252162456512, 2.5137205123901367, 1.9150186777114868, -0.08509179949760437, 0.07335860282182693, -1.9273062944412231, 0.2520577013492584, -0.9186553359031677, 1.2402271032333374, -1.2311217784881592, 0.20422431826591492, -0.062390413135290146, -0.3569563925266266, -0.4003745913505554, -0.3122195601463318, -1.5240541696548462, 0.5995477437973022, 1.2585855722427368, -1.0156769752502441, -0.7585468888282776, -1.253064513206482, 0.24470211565494537, 0.11376633495092392, -1.3101149797439575, -1.72486412525177, -0.14190854132175446, 1.0808608531951904, 1.3752336502075195, -0.3895503282546997, -0.38962069153785706, 1.8238861560821533, 1.419551134109497, -0.8965878486633301, -0.989684522151947, -1.7230662107467651, 0.49143263697624207, -0.330195814371109, -0.9048953056335449, -0.3653501868247986, -1.3306965827941895, 1.1798471212387085, -0.08967771381139755, 0.7065779566764832, -1.6369787454605103, -0.8827138543128967, 0.6572413444519043], [0.5688979625701904, -0.11750917881727219, -1.0150200128555298, -0.7312965393066406, 0.5528442859649658, 0.004776351619511843, -0.337612122297287, 0.6417142152786255, -1.5046929121017456, -0.7990649938583374, 0.02153613604605198, 1.5040756464004517, -0.6577517986297607, 1.2229238748550415, -0.44088634848594666, 0.854735255241394, 0.7745774388313293, 1.3506625890731812, 0.6358617544174194, 0.5600656867027283, -0.1689896285533905, 1.217393398284912, 1.1514263153076172, -0.3629404902458191, -1.741645097732544, 1.0914721488952637, 0.39496731758117676, -0.4955832362174988, -0.7303245663642883, 0.16983316838741302, -0.5634880065917969, 0.5954596400260925, -0.25581124424934387, 0.3867185115814209, -2.7940945625305176, -0.7531731128692627, -1.579795241355896, -0.8507612347602844, 0.39931389689445496, -0.06512415409088135, 0.4533911943435669, -0.6453508734703064, -1.4382553100585938, 0.3185591697692871, -0.719753623008728, 0.5961838364601135, 0.7761504650115967, 0.5233759880065918, -2.044419527053833, 0.31264108419418335, -1.1439011096954346, 0.897842526435852, -0.37191152572631836, 0.23723691701889038, 0.8707143664360046, -0.008719327859580517, 1.4424506425857544, -1.2584372758865356, -0.8724331855773926, 2.4495232105255127, 0.818223237991333, 1.7007948160171509, 0.46954309940338135, -0.524945855140686], [1.4943585395812988, 1.3859496116638184, 0.3656843304634094, -0.5679981708526611, 0.6360520124435425, 1.3802399635314941, -1.0064877271652222, -0.37931764125823975, -0.49375221133232117, 0.6938718557357788, 1.6112068891525269, -0.6629020571708679, 0.5224304795265198, 1.7981646060943604, -0.6019996404647827, 1.2324695587158203, -0.8681746125221252, -1.059667706489563, -0.2025073915719986, 1.0290621519088745, 0.04572214186191559, -0.8583725690841675, -1.4162273406982422, 0.6977763772010803, 1.9925379753112793, 1.6692208051681519, 0.04203606769442558, -0.7261157035827637, 0.44619086384773254, -1.0943267345428467, -0.8867244124412537, -0.7558334469795227, 0.5592957139015198, 1.9739632606506348, -0.7574946284294128, -0.9221000671386719, 1.2186360359191895, -0.8044843673706055, 1.353046178817749, -0.531865656375885, -1.1196763515472412, 0.115109384059906, -0.7663249373435974, -1.4434605836868286, -1.1690500974655151, -0.4480040967464447, 0.6453943848609924, 1.3806445598602295, -0.6948428750038147, -0.14362700283527374, 0.410291463136673, 1.4218358993530273, 0.051429346203804016, -1.1981643438339233, 1.1185364723205566, -0.9789149761199951, 0.19189682602882385, 0.5995882749557495, 1.2332159280776978, -0.42542779445648193, -0.6731253862380981, -0.5597110986709595, -0.85722416639328, 2.3743410110473633], [0.09817852824926376, 0.33089372515678406, -0.9017242789268494, 0.279610812664032, 0.5077676773071289, 0.10001464933156967, 1.4862741231918335, -0.17130224406719208, 1.3127951622009277, -0.31140777468681335, -0.8582549095153809, 0.33686861395835876, -1.8804869651794434, -1.327035903930664, -0.09305224567651749, -0.860805332660675, -0.22864720225334167, 0.733532190322876, 0.9935156106948853, 0.7794314026832581, 0.7016611099243164, -0.901523768901825, -0.35460761189460754, -0.8750571012496948, -1.1727598905563354, -1.1453853845596313, 1.8551961183547974, -0.8303028345108032, 0.16991718113422394, 0.7002075910568237, 0.38279834389686584, -0.7044036984443665, 0.17961134016513824, -1.5431348085403442, -1.2323243618011475, -0.21130584180355072, 0.7673062086105347, 1.0276182889938354, 0.7370584011077881, 0.24671146273612976, 0.47319960594177246, 1.4297953844070435, -0.9097868204116821, -0.4838130474090576, -1.0822988748550415, -0.8057454824447632, -0.66303551197052, 0.36074841022491455, -0.903481125831604, 0.5527318716049194, -0.7162327766418457, 1.208599328994751, 1.0143612623214722, -0.6364022493362427, -0.5184544324874878, -0.1063094213604927, 0.4910375475883484, -0.20879819989204407, 0.7772449254989624, -1.1250687837600708, -1.0731416940689087, 0.849738597869873, 0.6985753774642944, 0.20334933698177338], [-0.6620802283287048, -0.5730706453323364, 0.03538592904806137, -0.8739436864852905, -0.15587061643600464, -0.048891469836235046, 2.1790990829467773, -0.3007460832595825, -0.6727169156074524, -0.5264837741851807, 1.2350234985351562, -1.5482386350631714, 0.7112827897071838, 0.45029687881469727, 0.4980895519256592, 1.2091999053955078, -0.8207481503486633, 0.7706156969070435, -0.9289768934249878, 0.8383402228355408, 0.1318904459476471, 1.18648362159729, -0.38775256276130676, -1.2470431327819824, 0.16811025142669678, -0.20575930178165436, -0.3801710605621338, 0.7921314239501953, 0.5090847015380859, -0.42780253291130066, 0.5077677369117737, 0.666936457157135, 1.4955263137817383, 0.560651957988739, -0.2883415222167969, -1.4537112712860107, 1.1537424325942993, -1.1560792922973633, -0.9624719023704529, -0.08769858628511429, -0.2942228615283966, -1.4847935438156128, -2.2408082485198975, 0.472007155418396, 0.04020201787352562, 0.23302724957466125, 2.6821818351745605, -0.2150619626045227, 0.4156235456466675, 1.0855635404586792, 1.2391307353973389, 0.3678038716316223, -1.3621903657913208, 0.2967628538608551, 0.2205335795879364, 0.431503564119339, -1.6893913745880127, 1.7459454536437988, -0.3218272030353546, -0.2725972831249237, 2.2841601371765137, 2.3284242153167725, -0.1138525977730751, 0.6938652396202087], [-1.5313377380371094, -0.14434903860092163, -0.3100528419017792, -2.0346570014953613, 1.3752696514129639, -1.9336069822311401, 2.669997215270996, -1.4494987726211548, -2.5147669315338135, 0.0635421872138977, 1.1948732137680054, -1.9763400554656982, 0.10582486540079117, 0.050295159220695496, -0.8476631045341492, 0.03863675147294998, -0.13360798358917236, -0.17139476537704468, -0.38250118494033813, -0.7066915035247803, 0.34523507952690125, 0.6779600977897644, -1.1312785148620605, -1.2321171760559082, 1.9534449577331543, -0.4115036725997925, -0.8824042677879333, -1.768646001815796, 1.1059203147888184, 0.42086073756217957, -0.8888649344444275, 1.4526221752166748, 1.1055846214294434, -1.6256426572799683, -0.6045635938644409, 0.29445263743400574, 0.50991290807724, 0.921759843826294, -2.2590601444244385, -1.1493407487869263, 0.39625540375709534, 0.4590328633785248, 0.6908928155899048, -0.5617008209228516, 1.0803275108337402, -0.12230786681175232, 0.42945167422294617, 0.9230348467826843, -0.14984656870365143, 0.6615203022956848, -1.638252854347229, 0.7400050163269043, -0.12130674719810486, 0.11711329221725464, -0.5882457494735718, -3.097338914871216, 0.813178539276123, 0.6506825685501099, 1.6390595436096191, -1.7466803789138794, -1.781825065612793, 0.14213083684444427, -0.6058157682418823, -0.3609592616558075], [1.4323318004608154, 0.7512513399124146, 0.5924474596977234, 1.1745926141738892, -1.3091648817062378, -0.3628794550895691, 0.6658983826637268, -0.7406931519508362, 1.1506378650665283, -0.7294014692306519, 1.3443851470947266, -2.0875155925750732, 0.8008920550346375, -0.24247108399868011, -0.44119325280189514, 1.0293747186660767, 1.0320963859558105, -0.5896912813186646, -1.072398066520691, 1.1201918125152588, 0.3943588137626648, 2.145477771759033, 0.683037281036377, 0.5574367046356201, 0.12032121419906616, -1.047497272491455, -0.17631778120994568, 1.8107913732528687, 0.17675244808197021, -1.0245765447616577, -0.03140632435679436, 0.6935263276100159, -0.41132694482803345, -0.7475412487983704, -0.7960383892059326, 0.49656105041503906, 0.3263842761516571, 1.3564541339874268, 1.1271135807037354, 0.9369884729385376, -1.697670817375183, -0.005952952895313501, 0.46136611700057983, 0.22253109514713287, 0.6086367964744568, -0.5710323452949524, -0.26840120553970337, 0.3908405005931854, 1.226071834564209, 1.1291624307632446, 0.9716792702674866, -0.4441337287425995, 1.8576126098632812, 1.1773123741149902, 0.5504891276359558, 0.9419156312942505, -0.3630301356315613, -1.0084975957870483, -2.1579198837280273, 0.9621029496192932, 0.5304996371269226, -0.20262204110622406, 0.2995777726173401, -1.350258231163025], [1.5229328870773315, 1.6111146211624146, -0.6478345394134521, -0.5952853560447693, -1.180929183959961, 0.11376428604125977, -1.9019720554351807, -0.8296894431114197, -0.41305381059646606, 0.7210129499435425, -0.33188024163246155, 1.6360758543014526, 1.4564529657363892, 0.6549007296562195, -0.18007223308086395, 0.8085372447967529, 0.8764578700065613, 2.0512213706970215, -0.9600472450256348, 0.7018045783042908, 0.033134881407022476, 0.6989873647689819, -0.24011921882629395, -0.46202778816223145, 1.747852087020874, -1.0188277959823608, 0.5159691572189331, -0.5803763270378113, 1.3906632661819458, -0.1391303390264511, 0.24579183757305145, -0.7408368587493896, -0.7420985102653503, -2.100783348083496, 1.4571717977523804, 0.8626437187194824, -1.273447871208191, 0.5031960606575012, -0.8659777641296387, -0.1915934532880783, 1.0965425968170166, 1.2512080669403076, 1.443078875541687, 0.8904556632041931, -0.5764138102531433, -0.8496009707450867, 0.890821099281311, -0.1666364073753357, 0.0016675482038408518, -2.0962870121002197, -2.0646004676818848, -0.6799240112304688, 1.8145722150802612, -1.369937777519226, 0.6237272620201111, 1.9421323537826538, -0.41675177216529846, -0.6658526659011841, -0.14205294847488403, 1.4440124034881592, -0.12104585766792297, -0.4839646518230438, -0.3991847634315491, 0.30141133069992065], [2.3684751987457275, -0.5446182489395142, -0.6700097918510437, 1.4215961694717407, -0.559655487537384, -0.5201317071914673, 0.0106126070022583, -0.10589422285556793, -0.795382022857666, -0.4811779260635376, -0.8093088865280151, 1.533994436264038, -0.47339069843292236, 0.6294263005256653, -0.7920819520950317, -0.6913578510284424, -0.6392260193824768, 0.6939657926559448, 0.01445859670639038, 1.1621384620666504, 2.1208949089050293, 1.0330026149749756, -0.24970300495624542, -0.024208681657910347, -0.20064771175384521, -1.583141565322876, 0.19112852215766907, -2.236741304397583, 0.7816398739814758, 0.6169225573539734, -0.02442581206560135, 0.5657406449317932, -1.1765552759170532, -0.7629585266113281, 0.03470088168978691, -0.21661284565925598, -1.6693459749221802, 1.0703611373901367, -0.5966671109199524, -0.5375010371208191, -0.5978214740753174, -0.23653090000152588, -1.2951624393463135, 0.4981583058834076, 0.6651197075843811, 0.3790356516838074, 0.025634106248617172, 0.32540127635002136, 0.4395900368690491, 0.535282552242279, 0.9224082827568054, 1.040570616722107, 1.7611503601074219, 0.34864741563796997, -2.202754497528076, -1.6882834434509277, 0.5341277122497559, -0.8970386385917664, 0.6910337805747986, 0.7941861748695374, 0.7986721992492676, -1.4317141771316528, -0.2731330990791321, 0.6983967423439026], [-0.34710457921028137, 0.020176760852336884, -0.5874713659286499, 0.45921415090560913, 0.8414655327796936, -0.670353889465332, 2.159644842147827, -0.04972940683364868, 0.1521439552307129, -0.10102448612451553, 1.3312054872512817, 0.5918578505516052, -0.20944498479366302, -1.2884554862976074, 0.04213104769587517, -1.9214138984680176, 0.2680225670337677, -2.047983407974243, 1.005868911743164, 0.19277630746364594, -0.3098270893096924, 2.0919058322906494, 1.1699663400650024, -0.7442309856414795, -0.23393619060516357, 1.194966435432434, 0.38554301857948303, -0.1285402774810791, -1.6648998260498047, 0.7819913029670715, 1.5523760318756104, -1.4838582277297974, 0.25858786702156067, 0.2722376585006714, 2.6130130290985107, -0.15867258608341217, 1.4972349405288696, 1.3831311464309692, 0.6628785133361816, -0.5480559468269348, 0.7039471864700317, 1.0801948308944702, 1.3100123405456543, -0.9969757199287415, -1.3041446208953857, 1.4988011121749878, 0.3860509693622589, -1.36678946018219, -0.35657742619514465, -0.36513638496398926, -0.4942774772644043, -1.428418517112732, -0.796106219291687, -0.08413013815879822, 0.3525034785270691, 1.1941921710968018, -1.6576683521270752, -0.4337424337863922, -1.270621657371521, 1.0430867671966553, 0.04871524125337601, -0.24822275340557098, -0.2634138762950897, 0.9749042391777039], [0.4066787660121918, -0.41022422909736633, 0.23977039754390717, 2.1923375129699707, -1.1673918962478638, -0.00797722302377224, -0.7729182839393616, -0.32192444801330566, -0.578561544418335, 0.8065722584724426, -0.052181657403707504, 2.1425938606262207, -0.17818811535835266, -1.1826813220977783, -0.041667744517326355, 1.1177383661270142, -0.6478496789932251, 1.1941847801208496, -0.9677315354347229, -0.11438731849193573, 1.124487042427063, -0.6151832342147827, 0.3933275640010834, 0.8664956092834473, 0.5048004388809204, 1.9936885833740234, -0.6129202842712402, 1.2832386493682861, -0.17715021967887878, 0.48916515707969666, 0.41198256611824036, 0.3237835168838501, 1.117295265197754, -0.08623084425926208, -0.7218317985534668, 2.3356049060821533, 0.33601945638656616, 0.9284265041351318, -1.3149200677871704, 0.5316857695579529, 0.7784398198127747, -0.15123245120048523, -0.1667882204055786, -0.3679163157939911, -0.6770721077919006, 1.3377360105514526, -0.8488796353340149, 1.306244969367981, -0.6521270275115967, 1.7272192239761353, -1.0895313024520874, -0.6570799350738525, -0.2755638062953949, 0.3187102675437927, 0.4833146035671234, 0.5557458996772766, -2.4978139400482178, 0.47631222009658813, 2.489300489425659, -1.3662949800491333, 0.17564430832862854, -0.623250424861908, -0.6999706029891968, 0.4940539300441742], [0.6474714875221252, -1.458259105682373, -0.8381859064102173, -0.3710425794124603, -1.6690171957015991, 0.014500979334115982, 1.7682559490203857, -0.36216187477111816, 0.46851077675819397, -0.39909473061561584, -0.0848674327135086, 0.15711665153503418, 2.750246047973633, -0.6468931436538696, -1.746946930885315, 0.388652503490448, 0.8857260346412659, -1.0004948377609253, -1.8098112344741821, -0.9241867661476135, -0.7197577953338623, -0.9863176941871643, 1.0531669855117798, -1.8100579977035522, -0.24361418187618256, -0.6247897148132324, -0.24769896268844604, -0.7187947630882263, -1.9559406042099, 0.058708593249320984, -0.42018574476242065, 0.2411983460187912, -0.6186816096305847, 0.765960693359375, -1.0780318975448608, 2.5433642864227295, -1.145555019378662, -1.5251094102859497, 1.0468111038208008, 0.6078778505325317, -0.27432966232299805, 0.2660180330276489, -0.7052434086799622, 1.741920828819275, 0.16572165489196777, -0.4247802793979645, 1.0099595785140991, -0.10072405636310577, 0.6857130527496338, -0.21530455350875854, 0.7125222682952881, 0.4491751194000244, 0.2925553321838379, -0.7474978566169739, -0.1419709473848343, -0.15498727560043335, -0.20681273937225342, 1.5623929500579834, -0.7248813509941101, 0.6488111019134521, 0.24015355110168457, 1.0874630212783813, 0.286104291677475, 0.8777666091918945], [-0.5463201999664307, 1.3739720582962036, 1.054800271987915, -0.3960651159286499, -0.6952677369117737, -2.2164905071258545, -0.4102153778076172, 0.24591045081615448, -0.19533267617225647, -2.3046040534973145, 2.0516738891601562, 0.26952242851257324, 1.4775773286819458, 0.7110602855682373, 0.37699684500694275, -0.23777350783348083, -0.6736634373664856, -0.15380512177944183, -0.34410956501960754, -0.33927369117736816, 0.9638551473617554, 0.15494056046009064, 0.822485625743866, -0.997615396976471, -0.6516914963722229, -1.0046638250350952, -1.139099359512329, 0.8672753572463989, -0.046773653477430344, -0.916985273361206, 0.19797854125499725, 1.1969257593154907, -0.990719199180603, -1.4810082912445068, -0.22017335891723633, -0.06410744041204453, -1.6877992153167725, -0.47492048144340515, 0.7843524217605591, 0.19649924337863922, 0.4467700719833374, -0.5198322534561157, 0.28188371658325195, 0.17480146884918213, -0.34260252118110657, 0.3774731457233429, 2.374725580215454, -2.168184995651245, -0.7602468132972717, 2.6054446697235107, -1.2110389471054077, -1.059645175933838, 0.11728503555059433, -1.0746711492538452, 0.6168954968452454, -1.983120083808899, -1.1670562028884888, 0.6836426854133606, -0.4910469949245453, 1.1168850660324097, -0.22545191645622253, -0.023151863366365433, -2.1799747943878174, 1.5502985715866089], [1.0541706085205078, 0.2961692214012146, 0.525797963142395, -1.5518851280212402, 1.4857627153396606, 0.763570249080658, -0.6568708419799805, -0.3033944070339203, -1.8889753818511963, 0.2724611461162567, 2.8427984714508057, 0.3620487153530121, 0.9515774250030518, -1.6018849611282349, -0.8905314803123474, -0.3234977126121521, 1.0859897136688232, -0.31532710790634155, 0.24507199227809906, -0.8003293871879578, -1.078762173652649, -1.137813687324524, 0.7498624920845032, 0.3661361038684845, 1.1572999954223633, 1.037384033203125, -2.3955652713775635, -0.6755806803703308, -0.7903016209602356, 1.2096126079559326, 1.5445072650909424, 1.2291769981384277, -1.8304953575134277, -0.08726323395967484, -0.6446091532707214, 1.1568281650543213, 1.6011215448379517, -0.6751309633255005, 0.08684392273426056, 0.6811370849609375, -0.03007311373949051, -0.23178614675998688, -0.04813679680228233, 1.2830649614334106, -0.40201058983802795, 1.417435884475708, 0.48390260338783264, -0.5863320231437683, -0.3616847097873688, -0.02958529256284237, -0.8357441425323486, 0.4951636493206024, 0.2936406433582306, 0.6578715443611145, -0.2058636099100113, 1.0760610103607178, -0.14610476791858673, 1.1131384372711182, -0.757931649684906, -0.7147996425628662, -0.5921200513839722, 1.7942190170288086, -0.08907552808523178, 0.5700144171714783]], [[1.1089736223220825, -1.4966905117034912, 0.26267626881599426, -1.1504313945770264, 0.5260435342788696, -1.4305561780929565, -1.2190099954605103, 0.10373867303133011, 0.4356559216976166, 0.4561612606048584, 0.2516825199127197, 0.06111731007695198, -0.05763949081301689, 1.1308751106262207, 0.37980735301971436, 0.9276840686798096, -1.4127823114395142, 1.4554202556610107, -1.8397250175476074, -0.306706964969635, 1.1478244066238403, -0.6390500664710999, -0.8767624497413635, 0.14564037322998047, 1.5071772336959839, 1.3020455837249756, -1.5370947122573853, 0.40519464015960693, -0.4848625957965851, -1.338765263557434, -0.8611506223678589, -1.3161488771438599, -1.0379419326782227, -1.4677624702453613, 0.9860682487487793, -0.18763242661952972, 0.7609942555427551, 0.6059611439704895, -1.4948676824569702, 0.6895233988761902, -1.3509465456008911, -1.0561649799346924, 0.19055067002773285, -0.4862412214279175, -0.5373337268829346, 0.7266417741775513, 0.015275323763489723, -3.3660192489624023, 1.6658525466918945, 2.328258991241455, 0.6641473174095154, 0.9562974572181702, 0.37799787521362305, -0.9192537665367126, 0.07923293858766556, 0.26200422644615173, 0.38079994916915894, 0.45206424593925476, -0.38807693123817444, 0.7657690644264221, 1.9611879587173462, 1.4589953422546387, 0.5660702586174011, -0.6689872741699219], [-0.8076024055480957, 0.035615961998701096, -1.1270172595977783, 0.11520783603191376, 0.10455857962369919, 0.8073688745498657, 1.2907508611679077, 0.5133639574050903, 0.38884520530700684, 0.44267934560775757, -0.6165624260902405, 0.5415668487548828, 1.567541480064392, 0.9019259810447693, -0.09666869044303894, 0.3073421120643616, 0.961487352848053, 0.8831873536109924, 1.1343377828598022, -1.6114563941955566, 0.8919211626052856, 0.6837582588195801, 0.11574114859104156, 0.253122478723526, -0.6976677775382996, -0.415179967880249, -0.19000402092933655, -0.8743863105773926, -0.5307075381278992, 0.3762577176094055, -0.017546091228723526, -1.3900467157363892, 2.3491435050964355, -2.7422468662261963, 0.6098591685295105, -0.649608314037323, 0.28640016913414, 0.5807389616966248, 0.40314429998397827, 0.16653943061828613, -0.11807642877101898, 1.4532263278961182, -0.9525416493415833, 1.3220391273498535, -0.5026089549064636, 0.25946134328842163, 0.6554285287857056, -0.008382970467209816, -0.9693905711174011, -0.30868101119995117, 0.10214236378669739, -1.2323209047317505, 0.12811331450939178, -1.1923456192016602, -0.7562195062637329, 0.35392332077026367, -1.0573796033859253, 0.9889820218086243, 0.21624131500720978, -1.278340458869934, -0.9248111248016357, 0.8924088478088379, 1.7778574228286743, 1.1761131286621094], [-1.29912531375885, 1.0397562980651855, 1.8455660343170166, 1.5177619457244873, 0.45797300338745117, 1.0638383626937866, 0.5897607207298279, -0.3122597932815552, -0.614721953868866, -1.7305086851119995, 0.7111961245536804, -1.1889925003051758, 0.7068661451339722, 1.6285370588302612, 0.3880564272403717, -0.30012238025665283, -0.19282753765583038, 1.2023625373840332, -1.1577907800674438, 1.7606979608535767, -0.6707615852355957, -0.5122107863426208, 1.6570302248001099, -1.5560276508331299, -0.6073276400566101, 0.7571117877960205, 0.676876962184906, 0.41131842136383057, 0.023611335083842278, -0.8010706901550293, -0.023702245205640793, -1.20698082447052, -0.03840845450758934, -0.4900442659854889, -0.016301779076457024, 0.7302382588386536, -1.7679646015167236, 0.2685128152370453, -0.05490203574299812, -0.9616194367408752, 0.9513206481933594, -0.6213886737823486, -0.08354198932647705, -0.6447240114212036, -2.7731168270111084, 0.3686036169528961, -0.32158225774765015, 2.0450122356414795, -0.2747795581817627, -1.392232060432434, 1.3660892248153687, -0.5495470762252808, -0.6794924139976501, -0.4128171503543854, 1.8916252851486206, 1.830984115600586, -0.4616556763648987, 0.05579136684536934, 1.335412859916687, -0.40489867329597473, 0.4336412250995636, 0.04731478542089462, 2.0680291652679443, 1.6066677570343018], [-0.12214670330286026, -0.7919251322746277, 0.24324552714824677, -0.05126709118485451, -1.3562723398208618, 0.005699239205569029, 1.0110392570495605, -0.373879075050354, 0.008183377794921398, -0.7863246202468872, 0.04948268458247185, 0.16887503862380981, 0.2894389033317566, -1.611111044883728, -0.6587757468223572, -1.3861368894577026, -0.3405214548110962, -1.2300840616226196, 0.8675264120101929, -0.780735969543457, -0.0006725532002747059, 2.258315324783325, -0.550434947013855, 0.10967258363962173, 0.34237831830978394, 0.5245143175125122, 0.009562545455992222, 0.6920536756515503, -0.32493311166763306, 0.7356528043746948, 0.2446100264787674, 1.3066390752792358, 0.950234055519104, 0.10460943728685379, 0.09227383136749268, -0.6789006590843201, 0.7356429696083069, 0.20925860106945038, -1.6640199422836304, 1.4323770999908447, -1.4044110774993896, -1.191109538078308, 0.3429482579231262, -1.4778715372085571, 0.26543787121772766, 0.062443945556879044, -0.11493179947137833, -1.6052064895629883, 0.03821765258908272, -2.1469035148620605, 1.6418367624282837, -1.0616849660873413, -0.6762055158615112, 1.5797030925750732, 1.0260612964630127, -0.6350606083869934, -1.5072011947631836, 1.2646499872207642, -0.21572455763816833, -0.6246398091316223, 1.3761872053146362, -0.3702092468738556, 1.175845742225647, -2.0704312324523926], [-0.5546644330024719, 1.5097236633300781, 0.989252507686615, -0.01699831150472164, -0.4879954159259796, 1.1259669065475464, 1.6114673614501953, -1.0359935760498047, -1.6215298175811768, -0.0827474445104599, 0.31028980016708374, -0.16362448036670685, -1.5937552452087402, -0.6610833406448364, 0.4765526056289673, -1.2708698511123657, 1.0405027866363525, -0.12825217843055725, 0.3806227743625641, 0.011024721898138523, 0.21507461369037628, -2.079185724258423, -0.8283197283744812, -0.23902733623981476, -0.0002458461676724255, 0.08926176279783249, 1.0240895748138428, -1.0091017484664917, -1.524401307106018, -1.9127174615859985, -1.0173908472061157, -0.24509039521217346, 0.21540570259094238, -0.07634530961513519, -0.7934882640838623, 0.7244082093238831, -0.26726964116096497, 0.20690155029296875, -0.8830880522727966, 0.7379052639007568, -0.36337921023368835, 1.3179936408996582, 0.3532008230686188, 0.07871490716934204, -0.2355053871870041, -0.1569671779870987, -0.67653888463974, 1.1562159061431885, -0.37334078550338745, -1.3027399778366089, -0.631951093673706, 0.9922521710395813, 0.20326310396194458, 0.07214754819869995, -1.2179276943206787, 0.6859675049781799, -0.19596132636070251, -0.31891930103302, 0.22438156604766846, -0.5824577808380127, -0.8692101836204529, -2.0258045196533203, 1.3368855714797974, 0.6808653473854065], [2.0221283435821533, 2.3989109992980957, 0.5139979720115662, -0.5332359671592712, -0.2144361287355423, -1.1944025754928589, -0.6488991379737854, -0.05598565936088562, -0.7452701926231384, -0.04885204881429672, -0.32652562856674194, -0.0692167580127716, -1.778840184211731, 0.6877334713935852, -0.6659417748451233, -0.16863180696964264, 1.2406210899353027, -1.5894428491592407, 0.7149814963340759, 0.39176666736602783, -0.03418392315506935, -0.39394810795783997, -0.22981402277946472, -0.480991393327713, 0.626045823097229, -0.4373062252998352, -0.36754050850868225, -0.44221580028533936, 1.539339303970337, 1.8843706846237183, 0.8898724913597107, 0.5872921943664551, -0.9337216019630432, 0.520089864730835, 0.4344137907028198, -1.5092869997024536, 0.18399296700954437, -0.4151821434497833, -0.855750560760498, -0.09846324473619461, 1.7125366926193237, 1.0890134572982788, -0.803945004940033, -0.9618098735809326, 0.48896506428718567, -2.1333541870117188, 0.661486804485321, -2.037838935852051, 0.11291516572237015, 0.9608744382858276, -0.2596953511238098, 2.542245388031006, 0.5728853940963745, -0.32126978039741516, -0.8364658951759338, -0.899659276008606, 0.17062555253505707, 1.734087347984314, -0.866679847240448, -0.6859957575798035, 0.7364252805709839, -0.25666677951812744, -0.9248437881469727, -1.1556333303451538], [-0.7021802663803101, -0.3420594036579132, 0.9046680927276611, 1.218825340270996, -0.32920774817466736, -1.4810932874679565, -0.005283698905259371, 0.6381887197494507, -0.3106062114238739, -0.896312415599823, -1.53032386302948, 0.6576551795005798, 0.15625527501106262, 0.7129688262939453, 0.9949472546577454, 1.5265520811080933, 1.9553539752960205, 0.016024645417928696, -0.9081898331642151, 0.8374579548835754, 0.41405048966407776, 0.773463785648346, -0.6417446732521057, 0.05329355224967003, 0.008012956939637661, -0.10334249585866928, 0.46046164631843567, -0.2067149579524994, -0.9739088416099548, -1.5288375616073608, 0.495045930147171, -2.359707832336426, 3.2363545894622803, 0.7067136168479919, -0.5664986371994019, -0.5063830614089966, 1.203994631767273, -0.844879150390625, -0.44862428307533264, -1.1653703451156616, 0.7339956760406494, 0.9360461235046387, 0.42173656821250916, -0.20870374143123627, -0.449767529964447, -1.279356837272644, -0.7827801704406738, -2.0884222984313965, -1.062211513519287, -0.2951807379722595, 1.1569408178329468, 1.709661602973938, 2.146202325820923, -0.10039655864238739, -0.4980183243751526, -0.4907749593257904, 1.5999103784561157, 1.9393409490585327, -0.18298546969890594, -0.10296246409416199, 0.9512609243392944, -0.9428897500038147, -2.171659231185913, -1.2059201002120972], [-0.011062477715313435, -0.7841206789016724, -1.0266019105911255, 0.09260106831789017, -1.2727251052856445, 1.456926941871643, -0.976777970790863, -0.24847407639026642, -1.3730404376983643, 1.0799365043640137, 1.6537429094314575, -0.6362154483795166, -0.2862051725387573, 1.2168742418289185, 1.3973597288131714, 0.9107926487922668, -0.9409236907958984, -0.7888593673706055, -0.984201192855835, 0.5426540374755859, 0.01113828457891941, -1.060987114906311, 1.2288151979446411, -2.4340121746063232, 0.46341589093208313, -1.076485276222229, -0.5244843363761902, -0.6664490103721619, 1.6613496541976929, -1.6806291341781616, -0.8922966718673706, 1.217193603515625, -0.42195361852645874, 0.6670491099357605, 1.1689355373382568, -0.3345303535461426, 0.12009274959564209, 1.136976957321167, -0.215431809425354, 0.05270027369260788, -0.6429115533828735, 0.6383408308029175, 1.1192736625671387, 0.008204100653529167, 0.9137793779373169, -1.020473837852478, -0.7513718605041504, -1.1362870931625366, 0.7164668440818787, 1.42843759059906, 0.4686957597732544, 0.15129412710666656, -0.18734554946422577, 0.08301109075546265, -0.9413179755210876, -0.3923948407173157, -0.5148261189460754, -0.5352188944816589, -1.0999374389648438, 0.4812586307525635, 0.8269728422164917, 0.14311553537845612, 2.6706297397613525, 1.0589226484298706], [-1.0157755613327026, -1.3459742069244385, -0.054285384714603424, 0.41325777769088745, 0.47855880856513977, -2.10296630859375, 0.7229671478271484, 0.37420785427093506, 0.22652290761470795, 1.1214206218719482, 0.504697859287262, -1.0793484449386597, 0.19349007308483124, -1.613405466079712, 0.36535391211509705, -0.12834808230400085, 0.328194797039032, 0.6277376413345337, -1.6235758066177368, 0.7374956011772156, -0.5162360072135925, 1.1448551416397095, -1.0967495441436768, 1.1323422193527222, -0.6992267370223999, 0.7099541425704956, -0.022410690784454346, -0.31845295429229736, -1.384436011314392, -0.8206668496131897, -0.8127269744873047, -0.7749282121658325, -0.8170716166496277, 0.5106935501098633, -1.0577729940414429, -0.3127473294734955, -1.824333667755127, -0.5999399423599243, 0.5326266884803772, 0.3079003691673279, -1.652338981628418, -0.8246626853942871, -1.7032508850097656, -1.050635814666748, 0.4396180212497711, 1.3421090841293335, -1.164071798324585, 0.9230782389640808, -0.6621811985969543, 1.2893903255462646, -1.1731783151626587, 1.3767551183700562, 0.4936848282814026, 0.24040041863918304, 0.30660027265548706, 0.10542947798967361, -1.1230084896087646, 0.9433709383010864, -0.5692316889762878, -0.487691193819046, -0.3959517478942871, -1.0505534410476685, -2.297417640686035, 1.1313954591751099], [0.20132778584957123, 0.2713053822517395, 1.0227296352386475, 0.7589854598045349, -0.3471319079399109, -0.9155307412147522, -0.5328895449638367, 1.6799894571304321, 1.3928059339523315, 0.03803684934973717, -0.033467795699834824, 0.3421178162097931, 0.28348207473754883, -0.1762877106666565, -0.04286564514040947, -1.2248320579528809, 0.14237667620182037, 0.34293800592422485, -0.7064568996429443, -1.2033350467681885, -0.22864098846912384, 2.4388184547424316, -0.9891409873962402, -0.2388814091682434, -1.847397804260254, 0.2261841595172882, 0.27204057574272156, -0.34935787320137024, 0.010991865769028664, 3.4399802684783936, 1.0316797494888306, -1.3993767499923706, 0.5337685942649841, -0.6437699198722839, 1.1870055198669434, -0.6721002459526062, -0.13276337087154388, 0.9592581391334534, -1.1262385845184326, -0.21295776963233948, -0.21430283784866333, 1.084276556968689, -0.27498725056648254, 1.7941137552261353, -1.0436224937438965, 0.21530508995056152, -0.945967435836792, 0.18681225180625916, -1.2696894407272339, 0.9474520087242126, 0.3987595736980438, -1.5645256042480469, 0.950954020023346, 1.3115788698196411, -0.705739438533783, 0.3627854287624359, -0.2674642503261566, -1.1316241025924683, 0.44691652059555054, -1.3392024040222168, -1.0618855953216553, -1.6951614618301392, 0.7773047685623169, 0.23119863867759705], [-1.505828619003296, -2.056321382522583, 0.5692967176437378, -1.2146365642547607, 0.5391329526901245, 0.21094074845314026, 0.6532756090164185, -0.16641655564308167, 0.9230809211730957, -2.3979854583740234, -0.1652468740940094, -0.4293161928653717, 0.020481746643781662, 0.8421075344085693, 0.8436170220375061, 0.8215944766998291, 0.6078663468360901, 1.3580007553100586, 0.19374653697013855, 1.0436375141143799, 0.6533488035202026, 0.40472841262817383, -0.7519093751907349, -0.14277061820030212, -0.6181049942970276, 1.4971823692321777, -1.5388813018798828, 0.17171235382556915, -0.3766153156757355, 0.23532216250896454, -0.5046067833900452, -0.8893753886222839, -0.15741446614265442, -0.1029801070690155, -0.5809445977210999, -1.1853013038635254, -1.441489577293396, -0.8095394372940063, -0.4590688943862915, 0.3218116760253906, -0.8179781436920166, -0.8968713879585266, -1.7342413663864136, -0.6719807982444763, 1.7512024641036987, 1.0927025079727173, -1.6177111864089966, -0.23855066299438477, 0.11910141259431839, 1.3855228424072266, 0.2632395029067993, 0.3353530466556549, -0.3244927227497101, 1.2015570402145386, -1.363264560699463, -0.46090540289878845, 0.35716211795806885, -1.026268482208252, -0.6901109218597412, -1.545027732849121, 0.4188244342803955, -0.020115375518798828, 0.7979727983474731, -0.2960725426673889], [-1.3338589668273926, 0.863822340965271, 0.7207117080688477, 0.2542136311531067, 1.4196614027023315, -1.6174298524856567, -1.3606008291244507, 0.10539592802524567, -1.960525393486023, -0.4443836808204651, 1.4494776725769043, -1.7739595174789429, 0.6889544725418091, 1.9094455242156982, -0.23672030866146088, -1.0722678899765015, 1.5895073413848877, -0.536018431186676, 2.183490753173828, 0.8706063628196716, -1.8834725618362427, 0.6234064102172852, -1.6727186441421509, 0.9536249041557312, -0.5020712614059448, 0.07242783904075623, -0.05201675742864609, 0.6477634310722351, -0.04132191836833954, 1.6517119407653809, -0.22123964130878448, 0.06619357317686081, -1.5570787191390991, 0.46907684206962585, -1.21172034740448, 0.0480661541223526, -0.204656720161438, 1.2948673963546753, -0.33079782128334045, -0.821450412273407, 1.0349422693252563, -1.7747852802276611, 0.002489612437784672, 0.06489549577236176, 1.1891223192214966, -1.7447575330734253, -0.3633395731449127, -0.006755325477570295, -0.9823882579803467, -0.4565870761871338, -2.0928359031677246, -0.19518223404884338, 1.0888373851776123, -0.966323733329773, -1.7021323442459106, -2.5846619606018066, -0.8777419924736023, -0.5138559341430664, 0.05640915408730507, -0.9308044910430908, 0.015029214322566986, -0.7167624235153198, 1.1883279085159302, 0.34484365582466125], [-0.14427131414413452, 0.4281954765319824, 1.0313842296600342, 1.0995399951934814, -0.2784726023674011, -0.3008701205253601, -0.5071417689323425, 1.184777855873108, -0.06732378154993057, 0.9192821979522705, 0.49637287855148315, -2.538774013519287, -0.513709306716919, -1.6597590446472168, -1.1907787322998047, 1.3242385387420654, 0.4284838140010834, 0.2030823975801468, -0.17122462391853333, 0.4366151988506317, 0.35033491253852844, -0.8317424058914185, -1.089043378829956, 0.07095472514629364, -0.2612922787666321, -0.2402770221233368, 0.6560817360877991, -2.166518449783325, 0.3470277488231659, 1.300794243812561, -1.2280150651931763, 1.5333720445632935, 0.2840372323989868, 0.02558353915810585, -0.4879986047744751, 1.5358673334121704, -0.1544255018234253, -0.613797664642334, 0.568281352519989, 1.2503293752670288, -1.236205816268921, 0.47427040338516235, -0.9766901135444641, -1.1984232664108276, 0.3059501349925995, 0.4800780415534973, -0.5974692106246948, 1.2002507448196411, -0.3777756094932556, -0.03413034975528717, -0.6783639788627625, -2.064849615097046, 0.9031345844268799, -2.3059799671173096, -1.0042378902435303, -0.6471885442733765, -0.8668780326843262, 0.4076068699359894, -0.39279282093048096, 1.1185994148254395, -0.4463871717453003, -1.0860497951507568, 1.4349207878112793, -0.33448803424835205], [-1.8946144580841064, 0.016416503116488457, 1.0258817672729492, -1.1942996978759766, -0.3405992090702057, 1.1705322265625, 0.45887792110443115, 1.420261025428772, 0.7069233655929565, -1.2121278047561646, 0.27569064497947693, -2.4610936641693115, -0.7524440884590149, -0.052073609083890915, 1.2760374546051025, -1.6627732515335083, -0.34969156980514526, -0.2208816409111023, -1.1566526889801025, 2.0925774574279785, -0.5639041066169739, -1.013340711593628, -0.5352775454521179, 0.5909217000007629, 1.018847107887268, -0.8726481795310974, 1.530163288116455, 1.5988895893096924, 1.1231251955032349, 0.43271368741989136, 0.009401952847838402, 0.513226330280304, 0.66451495885849, 0.22248680889606476, 0.8409407734870911, 0.5722282528877258, -0.6691312789916992, 0.04964068904519081, 1.4770125150680542, 0.11843866854906082, -0.9195000529289246, 1.1722155809402466, -0.09430725127458572, -0.9325881004333496, -0.15118053555488586, 1.4713695049285889, 2.249786376953125, 0.12019067257642746, 0.11645280569791794, -1.2696030139923096, 0.8388094902038574, -0.07291559129953384, 0.6355388760566711, -0.1552562564611435, 0.2544555962085724, 0.0789242759346962, 1.9488670825958252, -1.5945801734924316, -1.2042903900146484, -0.6203284859657288, 0.17834527790546417, -0.22604532539844513, -0.41015857458114624, 0.9309251308441162], [-0.05460789054632187, -0.21955835819244385, 1.1537222862243652, 2.1953299045562744, -1.4260460138320923, 0.7101359963417053, -2.796689748764038, 1.393405795097351, 1.8198052644729614, -1.131363868713379, -0.08085572719573975, 0.7112930417060852, 0.2646885812282562, 0.2493402361869812, 0.9017235040664673, -0.6814718246459961, 0.9111030697822571, 0.88767009973526, 0.49720004200935364, 0.24057339131832123, -0.3599846065044403, 1.2129426002502441, 2.3040759563446045, 0.3390972912311554, 0.7244102358818054, 1.9356589317321777, -0.10496463626623154, -0.4038081765174866, 0.49761736392974854, -0.2846870720386505, 0.1379123330116272, 1.3227920532226562, 0.4360351860523224, -1.2379382848739624, 0.6668266654014587, 1.0500131845474243, 0.8525663614273071, -0.9995918273925781, -0.326080322265625, -0.11338987201452255, -2.088859796524048, -1.2732688188552856, 1.5785857439041138, 0.2937629520893097, 1.7744885683059692, 1.0478551387786865, -0.6667283177375793, 1.066428780555725, 0.045834168791770935, 0.5733726024627686, 0.7615579962730408, -0.7881637811660767, -0.9948196411132812, -0.9104479551315308, 1.1762800216674805, -0.35481584072113037, 0.0908958911895752, -1.4988062381744385, 0.75360506772995, -0.8101924657821655, 0.6472945213317871, -0.4191817343235016, 0.994048535823822, 1.0847359895706177], [2.0084805488586426, 0.8239649534225464, 0.4596250653266907, -1.277248501777649, 0.9399021863937378, -1.0534101724624634, -0.72225022315979, 0.21817976236343384, -0.6072782874107361, 1.7814607620239258, -1.1104204654693604, 0.23513199388980865, 0.28677645325660706, 1.7890305519104004, 0.9724376201629639, 1.1428756713867188, 0.4018353819847107, 1.0142242908477783, 1.5308815240859985, -1.070124864578247, -0.924452006816864, -0.07550151646137238, 1.0186148881912231, 0.13910485804080963, -1.7607356309890747, 0.16388604044914246, -1.397768497467041, 0.06299221515655518, -2.546806573867798, -0.20776477456092834, 0.9298987984657288, 0.3447285294532776, -0.670768141746521, 1.5762829780578613, -0.5481013059616089, -0.08106397092342377, 0.6225740313529968, -0.5099205374717712, 0.3079323172569275, 2.0524306297302246, -0.46192672848701477, -0.6605169773101807, 0.6368751525878906, -1.0939756631851196, -0.21611396968364716, -1.6795365810394287, 0.8811679482460022, -0.5915853381156921, -0.2534053325653076, -1.3825361728668213, -2.244352340698242, -0.2145938277244568, 0.3120184540748596, -1.3462501764297485, -0.15002460777759552, -0.3501906991004944, 1.2764954566955566, -1.2839815616607666, -0.4812203049659729, 1.96257483959198, 0.058710455894470215, -0.5375595688819885, 0.3475242853164673, -0.5571439266204834]], [[-0.31126755475997925, -1.633825659751892, -1.312991976737976, -0.3064867854118347, -0.6712002158164978, 0.3130292296409607, 1.398019552230835, -0.1605917066335678, 0.10965917259454727, 0.37676534056663513, -0.10623137652873993, -1.7100275754928589, -0.7369678020477295, 0.16082151234149933, 0.0062384526245296, 1.3226375579833984, -0.3906497061252594, 1.778928279876709, -1.8089698553085327, -0.6414799094200134, 0.4513486325740814, -0.04488536715507507, 0.6636821031570435, -0.1507962942123413, -0.48134130239486694, -0.708073616027832, -0.42193326354026794, 1.471052646636963, 1.073845624923706, 0.20087578892707825, -0.5821544528007507, -0.17455339431762695, -1.6029415130615234, 1.1085618734359741, 0.030785439535975456, 0.8106206655502319, -0.9862696528434753, -0.6860988140106201, -0.8885257840156555, 0.490283727645874, 0.05405588820576668, -0.34063178300857544, -0.6003024578094482, -0.16988934576511383, 0.5017129182815552, -1.2304010391235352, 0.02086555026471615, 1.108886480331421, 0.07600197941064835, -0.9753072261810303, 1.0087952613830566, -2.2167177200317383, 0.19018061459064484, -0.5259542465209961, -0.38011908531188965, 0.55415278673172, -1.5536835193634033, -0.08138252794742584, 1.020307183265686, -0.2878114879131317, 1.505366563796997, -1.5877320766448975, -0.08972834050655365, -1.1518056392669678], [0.699259340763092, 0.1458098441362381, -0.4377431869506836, 0.27056393027305603, 0.9945642352104187, -0.3267586827278137, 0.2447846233844757, -0.7891567945480347, -0.12771129608154297, 1.5143953561782837, -0.284902960062027, 0.12701737880706787, 0.46284541487693787, 0.15641839802265167, 1.1462332010269165, 0.001966227311640978, -1.337554693222046, -1.2519383430480957, 0.23224832117557526, -0.8310104012489319, -1.4724394083023071, -0.2650283873081207, 1.0401768684387207, 0.34567439556121826, -2.3058054447174072, -0.3958854675292969, 0.39053648710250854, 0.08969991654157639, -2.1597890853881836, 0.46926140785217285, 0.5970431566238403, 0.8529700040817261, 0.930584192276001, -0.7358877062797546, -0.4999230206012726, -0.8559557795524597, -0.17442204058170319, -0.1888599395751953, 0.1307845413684845, 0.30117231607437134, -1.1680485010147095, -1.0227863788604736, 0.5149991512298584, 1.0780085325241089, 0.1908871978521347, 0.43344417214393616, -0.025130445137619972, -0.6944199204444885, 2.2615485191345215, 1.2120351791381836, 2.6029815673828125, 0.6470953226089478, -1.3176614046096802, 0.7777363061904907, -0.7859163284301758, 0.5525839328765869, 0.6112077832221985, -0.6710413694381714, 0.7625138759613037, 0.29259854555130005, 0.5015050172805786, 0.03896922990679741, 2.3291499614715576, -0.1390099823474884], [-1.0944712162017822, -0.2605668604373932, 0.31403642892837524, 0.7506927847862244, -0.19885389506816864, -1.3111640214920044, 0.24171580374240875, -0.048057712614536285, 0.3525422215461731, 0.2993653118610382, 1.2221962213516235, 0.8022392988204956, 0.8032981157302856, 0.133571594953537, -1.7289382219314575, -0.0072916592471301556, 0.2354741096496582, -1.582431435585022, -1.043169379234314, -0.50648432970047, -0.8721634149551392, 0.3299238085746765, 0.28020766377449036, 0.27363041043281555, -1.2591955661773682, 1.0985912084579468, -0.6193689107894897, -0.4166891276836395, -0.6791438460350037, -0.15335498750209808, -0.5933610796928406, 0.025438042357563972, -0.5932837724685669, 1.1002768278121948, -2.36734676361084, 0.44871264696121216, 1.1164554357528687, -0.10735362023115158, -0.6461482048034668, -1.810980200767517, 0.41294530034065247, -0.6039702892303467, 1.2623668909072876, 0.4646873474121094, -0.9801568984985352, 0.697616696357727, 0.5839145183563232, -0.45476871728897095, 2.205500841140747, 1.8334287405014038, -2.2093124389648438, 0.02921815775334835, -1.0932695865631104, -0.012302037328481674, 0.8345724940299988, 1.2677891254425049, -1.4890493154525757, 1.5859774351119995, 0.6061033606529236, -3.7490406036376953, 0.6888012290000916, 0.6728232502937317, 1.6374350786209106, 0.6904560327529907], [-1.815544843673706, 0.1558510810136795, 1.5179327726364136, -1.3619842529296875, 2.0486361980438232, -0.09868750721216202, 0.5933120250701904, 0.7429192662239075, -0.6801642775535583, -3.0283524990081787, -0.8458359241485596, -0.5416452288627625, -0.47029468417167664, 0.2813439965248108, -0.7280969619750977, -0.7657070755958557, 1.1010452508926392, -0.505820095539093, -0.7345246076583862, 0.5798590183258057, 0.07269231975078583, -0.42672184109687805, 0.3545769453048706, 1.3417786359786987, -1.9743469953536987, -0.5595939755439758, -0.03005685843527317, 0.45909175276756287, 0.15215323865413666, 1.0407580137252808, 1.6737910509109497, 0.8221412301063538, -0.5936598181724548, -0.09206240624189377, 1.3759515285491943, 0.15144644677639008, -1.2671422958374023, -0.47585880756378174, 1.447153925895691, 1.74116051197052, 0.7161998748779297, 0.28574076294898987, 0.10595621913671494, 0.7877230644226074, -0.5909456014633179, 0.004836553242057562, 0.8606547713279724, -1.929835319519043, 0.12608186900615692, -0.07076389342546463, 0.3359331786632538, 0.14325641095638275, 1.9586782455444336, -1.6403746604919434, 0.6990482211112976, 0.4012935757637024, -0.1595035195350647, -0.6368409991264343, -1.8567430973052979, -0.4640183746814728, 0.542281448841095, -0.6475242376327515, -0.7182794213294983, 1.2683134078979492], [0.5535072684288025, 0.28800031542778015, 0.48885267972946167, -0.6195751428604126, 0.6402856707572937, -0.3676314949989319, -1.907313346862793, -0.3816147446632385, 1.1331347227096558, 1.9562530517578125, -0.7539899945259094, -0.8243610262870789, 1.7262451648712158, -0.3073180317878723, 1.352034091949463, -0.03684188425540924, -0.20406125485897064, -0.012875836342573166, -0.8071743845939636, -1.8698890209197998, 0.4884239137172699, 0.5237534046173096, -0.502406895160675, 0.16870561242103577, 1.4422905445098877, -0.2942466139793396, 0.06107575446367264, 1.8155235052108765, -0.16931036114692688, 1.7104347944259644, 0.6318448185920715, -1.1047217845916748, -0.9108427166938782, 0.7697288393974304, -2.500281572341919, 1.528267741203308, -0.9184139370918274, -0.7492311596870422, 0.9036695957183838, 0.5977103114128113, 2.074956178665161, -0.579376220703125, 0.046277280896902084, 0.5725021958351135, 0.13288578391075134, -1.3721132278442383, -0.7021996378898621, 0.8208670020103455, -0.9493730068206787, 0.04545307531952858, -0.25722986459732056, -1.4086544513702393, 0.10423202067613602, 0.38345423340797424, -2.576920747756958, -0.5923125743865967, 1.1181156635284424, -0.8926628828048706, 1.474948763847351, 1.2727060317993164, -1.9143880605697632, -0.5259888172149658, -0.16595353186130524, -1.2183376550674438], [1.8236336708068848, 0.2444811910390854, -0.7244527339935303, -1.1772385835647583, -0.43615174293518066, -0.4460369050502777, -0.8778502345085144, -0.7656209468841553, 2.115889310836792, 0.12138540297746658, -0.7652015686035156, 0.9164065718650818, 1.0412912368774414, 0.026010382920503616, -0.3168112337589264, -0.4500276744365692, -0.8047758340835571, 1.0879764556884766, 2.848478317260742, -1.8502001762390137, -0.26037007570266724, 0.9001337289810181, -0.7495353817939758, -0.9005469083786011, -1.7853453159332275, -2.1280264854431152, 1.3059672117233276, 1.1773265600204468, -0.027085741981863976, 1.6865296363830566, 0.2355867326259613, 2.3119735717773438, 1.1848127841949463, -0.7597147822380066, -0.17614173889160156, 0.2536540627479553, 2.068742513656616, -1.331412672996521, -0.18855495750904083, -0.9066638350486755, -0.7669628262519836, 1.8028295040130615, 2.5876100063323975, 0.10221115499734879, -1.0634839534759521, -1.4806050062179565, 1.598171353340149, 1.2552459239959717, -0.43421098589897156, -0.0403442308306694, -0.3647333085536957, 1.612173080444336, 0.0111728236079216, 0.2640262544155121, -0.28455841541290283, -1.1346415281295776, -0.6331868767738342, -0.2610613703727722, -0.5579301714897156, -0.008703036233782768, -1.3479896783828735, 0.9955010414123535, -1.386734962463379, 0.010809101164340973], [0.46750956773757935, 1.3877149820327759, -2.000490427017212, -0.5948857665061951, 1.91456937789917, 0.5921320915222168, -0.097413070499897, -0.7278419733047485, 0.7108716368675232, 1.6490446329116821, -0.07254356890916824, -0.8014733791351318, -0.18582895398139954, -2.046112298965454, -0.9556812047958374, -0.28377246856689453, -0.31295594573020935, 0.6547421216964722, 0.6184942126274109, -0.7300668954849243, 1.6041462421417236, -0.12017872929573059, -0.7485383749008179, 0.054123010486364365, -0.837454617023468, -1.6279027462005615, -1.1326392889022827, -0.11989687383174896, 1.0748822689056396, -2.058478832244873, -0.6260658502578735, 0.6669113636016846, -0.7024955749511719, -0.04049283266067505, 0.13573943078517914, 1.0023761987686157, 0.11143721640110016, -0.1760084182024002, 2.2418813705444336, -0.6301291584968567, 0.2702856659889221, -3.3652865886688232, 0.0026107411831617355, -2.387458086013794, -0.47846370935440063, -2.947700262069702, -0.15035167336463928, 1.0192054510116577, -0.4044959843158722, -0.16882352530956268, 1.2531366348266602, -0.561224102973938, 0.3298380672931671, -1.4393746852874756, -0.9334384202957153, -2.357325792312622, 0.40810754895210266, -1.041817307472229, 0.10881348699331284, 0.4216797947883606, 0.435268759727478, -0.6838092803955078, 0.30300694704055786, -0.18322736024856567], [-0.9655721783638, -1.590694546699524, -0.028370872139930725, -0.22586044669151306, 0.00943665113300085, -0.0074063437059521675, 0.12288948148488998, 0.8184366822242737, 1.4742602109909058, -0.41458648443222046, -0.6745707988739014, 0.5878909826278687, 0.08446675539016724, -0.35319826006889343, -0.03852174058556557, -0.7199504375457764, 0.7827072739601135, -0.5524682402610779, 1.2341996431350708, -0.9681223034858704, -0.19985130429267883, 0.8093665838241577, 0.8124493360519409, 0.09969866275787354, 0.03921542689204216, 1.3329675197601318, -0.9282236099243164, -2.5573458671569824, 1.3629436492919922, -0.5731582045555115, -0.8185078501701355, -0.014836634509265423, 0.8095337152481079, 0.028647564351558685, -0.10401997715234756, 0.15486203134059906, 1.5734350681304932, -0.007412651088088751, 0.8269698023796082, 0.8285654783248901, 0.7029123306274414, -0.32353395223617554, -0.8679145574569702, 1.6798771619796753, 1.7559894323349, 0.3464139401912689, -1.4221758842468262, 0.5222637057304382, 0.6271716952323914, 0.08894725888967514, -1.1883749961853027, 0.20364879071712494, -0.9378261566162109, -0.5856233239173889, 0.37563827633857727, -0.2472139298915863, 0.4682615101337433, -0.47030961513519287, 0.21322070062160492, 0.10997350513935089, 0.0862070843577385, 1.482102870941162, 0.9574704766273499, 0.2869165539741516], [0.9291006326675415, 2.301206350326538, -0.7692360281944275, 1.4729797840118408, -0.9485248923301697, -2.5273308753967285, -0.5877767205238342, 2.14471173286438, 1.0879539251327515, 1.025308609008789, -1.500324010848999, 3.1464920043945312, -0.33032867312431335, -0.2794433832168579, 1.7371057271957397, 0.2191193550825119, -1.5567030906677246, 0.17291763424873352, -0.4505856931209564, 0.5966894030570984, 0.1822904795408249, 0.44663766026496887, -0.7675304412841797, -2.368989944458008, -0.8399706482887268, -1.5619657039642334, -0.3307429254055023, -0.04488462954759598, -1.80880606174469, -0.5602694153785706, 1.6523995399475098, 0.7262890338897705, 0.7160658836364746, 0.4052943289279938, -0.8775001168251038, 2.1110713481903076, -1.0928537845611572, -0.1426444798707962, -0.33683153986930847, 0.2716589868068695, -1.3535643815994263, -1.4780998229980469, -2.216350793838501, -0.6707425713539124, -1.5930782556533813, -0.9200772047042847, 0.657149612903595, -0.8489448428153992, -0.010087192058563232, 0.22927263379096985, 0.5671122670173645, -0.7254976630210876, -0.11600062251091003, 0.0818478986620903, -1.6501520872116089, -1.0937612056732178, 0.7662957906723022, -0.5974608659744263, -0.2283601313829422, -0.1405850499868393, -1.4171398878097534, -0.05122203752398491, -0.7755612134933472, 0.9980236291885376], [1.5692973136901855, 1.3887523412704468, 1.1607857942581177, 0.781565248966217, 0.06970545649528503, 0.09331757575273514, 1.1436747312545776, -0.3316820561885834, 1.378421425819397, -0.06500054895877838, -1.085910677909851, -0.7342050075531006, -1.3485933542251587, 1.74961256980896, 0.5393688678741455, 0.5913384556770325, -0.2823488414287567, -1.527956485748291, -1.0251696109771729, 0.4379808008670807, -1.4372261762619019, -1.3172143697738647, 0.9675083756446838, -0.8570849299430847, 1.0858144760131836, -0.9099048376083374, -0.7404391169548035, -0.6306585073471069, 1.4338361024856567, 1.8775969743728638, 0.28927507996559143, 0.8441290259361267, -1.1951406002044678, -0.560271143913269, -0.17422239482402802, -1.155376672744751, -0.20626911520957947, 1.1461769342422485, 0.7745732665061951, -0.1760801374912262, -1.1676619052886963, 0.23065021634101868, 0.6584824323654175, 1.324562668800354, -0.009877345524728298, -0.35276198387145996, 0.333858847618103, 1.6290562152862549, -1.2063244581222534, 0.07120299339294434, 1.7576583623886108, 0.8372432589530945, -1.6816647052764893, -2.4429633617401123, 0.9923718571662903, -0.1392795890569687, 0.35784032940864563, 0.864284336566925, 1.2651978731155396, 0.4169448912143707, -0.736356794834137, 0.014938847161829472, 1.2742940187454224, -1.479832649230957], [-1.5120675563812256, 0.23943892121315002, 0.4824756979942322, -0.6477785110473633, 1.4282217025756836, 0.3094874620437622, 1.00900137424469, -0.1410733014345169, 0.5262024402618408, -0.5146346092224121, -0.6862275004386902, -1.1825922727584839, -0.11405078321695328, 0.4842548072338104, 0.6807953119277954, -0.24633102118968964, -1.9999308586120605, 0.8394330143928528, 1.1693825721740723, 0.33260104060173035, 1.5693304538726807, 1.5369880199432373, -1.2892580032348633, 1.6853958368301392, -0.34591054916381836, -0.5281897187232971, -0.6560148000717163, -0.3032620847225189, 0.4496411681175232, 0.05016026645898819, 1.7226959466934204, 0.13873934745788574, 1.4046608209609985, -1.4269105195999146, -0.7834709286689758, 0.21880877017974854, -0.2727666199207306, 1.890929102897644, -1.4038357734680176, -0.6597940325737, 0.7621568441390991, 0.44876790046691895, -0.6442002654075623, -1.109034538269043, 0.12346245348453522, -0.10091224312782288, 1.1213610172271729, -1.0764154195785522, -1.0310660600662231, 1.2066861391067505, -0.6615270972251892, 1.533352255821228, -1.0354357957839966, 0.26033535599708557, 0.5908212065696716, -1.1993037462234497, 0.30921870470046997, 0.16926367580890656, 0.29962965846061707, -0.719188928604126, -0.5982891917228699, 0.20739148557186127, -1.094519853591919, -0.32133734226226807], [-1.0162408351898193, 0.3889338970184326, 0.4402914047241211, 0.7248733639717102, 0.43394312262535095, 0.2405143529176712, 2.0858137607574463, 2.72709584236145, 1.053209900856018, 0.4582694470882416, 0.951450526714325, 0.019158393144607544, -1.153706431388855, -0.6382384300231934, 0.6011850237846375, -1.058443307876587, 0.8754989504814148, -0.9247841835021973, -0.8746116161346436, -0.8470256328582764, -0.10827770084142685, -1.2436519861221313, 1.8226133584976196, -0.7801300287246704, 0.8913984298706055, 0.3572287857532501, 0.031485751271247864, 0.5445228219032288, 1.0412896871566772, -0.47616147994995117, -0.4928244650363922, 1.054093837738037, -0.3381257951259613, -0.5881092548370361, -0.7155128717422485, 1.1397370100021362, 1.626592993736267, 1.0090410709381104, -0.40096673369407654, 2.1137709617614746, -1.2331072092056274, 0.9709633588790894, -0.7204899191856384, -0.7832475900650024, 0.9775804877281189, -1.2330280542373657, 0.08374620229005814, 0.5345967411994934, 0.3732927739620209, 0.631066620349884, 0.48999106884002686, -0.35587623715400696, -0.16379621624946594, -1.5446091890335083, -0.5856989622116089, 0.024182746186852455, -1.3868404626846313, 0.11712290346622467, -0.37938588857650757, 0.2557627856731415, 0.39315131306648254, -1.639596939086914, -0.18939058482646942, 1.2990870475769043], [-0.44422706961631775, -0.42029938101768494, -0.02605525031685829, -0.08726196736097336, 1.2723166942596436, 1.758579969406128, -1.229793667793274, -1.0023173093795776, 1.9679933786392212, -0.24353191256523132, 0.6416472792625427, 0.12230533361434937, 0.24876543879508972, 0.7575832009315491, -1.3405390977859497, 0.04046347364783287, 1.327630639076233, -0.17602911591529846, 0.025003086775541306, 0.5300042033195496, -0.47122061252593994, 1.3536044359207153, 0.31436851620674133, 1.0328766107559204, -1.006130337715149, -0.20612601935863495, -1.142632246017456, -1.2716771364212036, 1.6150591373443604, -0.0684816837310791, -1.4400055408477783, -0.6923975944519043, -0.6617415547370911, -1.1136634349822998, 0.08763064444065094, 0.8639155030250549, 1.3053600788116455, -0.8667987585067749, 0.5344215035438538, 1.4746246337890625, 0.8614041209220886, 0.042319390922784805, -0.8988246917724609, 2.058741331100464, -0.1671382188796997, 0.4229089915752411, -0.7846723794937134, 0.5170933604240417, -0.4501166343688965, -0.8278093338012695, -0.051614757627248764, -0.007211350370198488, 2.1410012245178223, -0.07486838847398758, -1.251551628112793, 1.0156925916671753, 0.36999398469924927, -0.10908389091491699, -0.7778695225715637, 0.6304951906204224, -0.3586026728153229, 0.6077878475189209, 0.3705030381679535, 0.6492236852645874], [-0.23660719394683838, 3.141127824783325, -0.09082786738872528, -1.5457547903060913, 0.3115004897117615, 0.4611653685569763, -0.36391887068748474, 0.5727981328964233, -0.3193208873271942, 0.36772775650024414, 1.1316618919372559, -0.16728536784648895, 1.0109219551086426, 0.8475689888000488, -2.3331267833709717, -0.9326637983322144, -0.5152154564857483, -0.05297510325908661, -0.18972371518611908, -0.19330133497714996, -1.2524712085723877, 1.1590523719787598, -0.536984920501709, -0.17953503131866455, 0.9447252154350281, -1.0890557765960693, -1.4330288171768188, 0.7053103446960449, -0.25481224060058594, -1.659636378288269, 0.34304943680763245, -0.41573700308799744, -0.3922463059425354, 1.391624927520752, 0.28503915667533875, 0.02492232620716095, 0.5936241149902344, -0.8948531150817871, 0.38787856698036194, -0.01852712780237198, -0.1587294191122055, -0.05192289128899574, -0.8909884095191956, -0.808718740940094, 1.314632773399353, 1.9673285484313965, -1.058344841003418, -1.599233627319336, 0.8708231449127197, -0.4685148000717163, 0.17656926810741425, 0.6088211536407471, 0.6057928204536438, 0.26236310601234436, 1.3032386302947998, 0.6696351170539856, 1.1931750774383545, -0.3553939461708069, 0.27188268303871155, 0.19292917847633362, 1.2637170553207397, 0.27978986501693726, 0.886006772518158, -0.03933180123567581], [0.46787965297698975, 0.6482694745063782, -0.1528944969177246, -1.2093150615692139, 0.812518835067749, -1.0625450611114502, -0.12214308232069016, -1.2209341526031494, 1.445733904838562, 0.6021465063095093, 0.3635264039039612, -1.3230795860290527, -0.7512364983558655, 0.1964588761329651, 0.33928728103637695, 0.9349398612976074, 0.31222954392433167, 0.06774821877479553, -1.0907374620437622, -0.25866055488586426, -0.9080476760864258, 1.044511079788208, -0.5131070017814636, 1.222408413887024, -0.2517062723636627, -0.02987331710755825, 1.541212558746338, -1.5747148990631104, 0.9313893914222717, 0.2193046361207962, -1.3352508544921875, 0.6001631617546082, -0.10297916829586029, -1.6550768613815308, 0.4531874358654022, 1.3953533172607422, 0.9087499976158142, -2.304154634475708, -0.5796934366226196, 0.3081700801849365, 0.12562718987464905, -0.18706925213336945, 0.47169211506843567, -0.6766178607940674, 1.7425824403762817, -0.14881569147109985, -0.7449934482574463, -0.7953739166259766, -1.7154840230941772, -0.10622920840978622, 0.6843132972717285, 1.9290046691894531, -0.515913188457489, -1.188791275024414, 1.029227375984192, 1.070327877998352, -0.5244888663291931, -0.3278094530105591, 1.8233851194381714, 0.7079766988754272, 1.259814739227295, 0.3585486114025116, 0.882588267326355, 0.30328187346458435], [-2.2820851802825928, 0.44820863008499146, 1.4160405397415161, -0.022318322211503983, -2.182304859161377, -1.352728247642517, -0.8872042894363403, -0.5304996967315674, -0.3127555251121521, -0.2877546548843384, -0.5370592474937439, 0.029422592371702194, 0.27520033717155457, 0.2474212795495987, 2.195962429046631, -0.20317067205905914, 1.0732600688934326, -1.0423176288604736, 0.5445671677589417, -0.7417594790458679, 0.6199212074279785, 0.5446746945381165, -0.5425122976303101, -1.2720372676849365, 1.5949980020523071, -0.49779820442199707, -0.5080084204673767, 0.039243459701538086, -0.653679609298706, -0.5775099396705627, 1.639513373374939, -0.27606886625289917, -0.3025437593460083, -0.22928868234157562, -0.6552258729934692, -0.492587685585022, -0.9644147157669067, -0.39705419540405273, 0.032705195248126984, 1.4220342636108398, 0.8290674686431885, -0.525603175163269, 0.20035982131958008, 0.3255347013473511, 0.08541785180568695, -0.35189396142959595, 1.0026289224624634, 1.2849174737930298, -1.982796549797058, 0.16865168511867523, -0.15619522333145142, 0.2708267271518707, 1.2579092979431152, 1.8139309883117676, -0.02406446821987629, 0.09537358582019806, -1.3855427503585815, -0.8324429988861084, 0.9775243997573853, -0.46951693296432495, -0.5805636048316956, -1.4031047821044922, -0.9190128445625305, -2.0233800411224365]]]], "dtype": "float32"}, {"__nd__": [[[[1.8236860036849976, -0.4948176443576813, -1.6548939943313599, 0.8525108695030212, 0.23392602801322937, -0.09739526361227036, 1.1184484958648682, 0.6407340168952942, -0.5115337371826172, 0.014273080043494701, 2.3764126300811768, 0.4918953478336334, -0.4286913275718689, 1.2802897691726685, 0.6752033829689026, -0.24169336259365082, 1.0262638330459595, -1.4908397197723389, -0.14501754939556122, 0.768665075302124, -2.1209254264831543, -2.6070568561553955, -0.5639466643333435, -0.37381288409233093, 0.20886926352977753, 0.8504350185394287, -0.41378721594810486, -1.1666158437728882, -1.665786862373352, 0.6232264637947083, 0.6302474141120911, 0.763708770275116, 0.017941638827323914, 0.36980846524238586, -0.6489982604980469, -0.1855204850435257, 1.5768555402755737, 0.6897691488265991, 0.9889079928398132, -0.8455129265785217, 0.8628105521202087, 0.7188106775283813, -0.43376046419143677, 1.6524296998977661, 2.1344106197357178, -0.2881642282009125, -0.8555700778961182, -0.5740684866905212, 0.36849454045295715, -1.5388137102127075, -2.040254592895508, 1.1418769359588623, -1.2641302347183228, -0.5077762603759766, 0.4596583843231201, -0.24508202075958252, 2.36586856842041, 0.34485459327697754, 0.5177354216575623, -1.006761074066162, -0.3329792618751526, 1.065652847290039, 1.187278151512146, 0.10287303477525711], [0.4139866828918457, 0.03797227889299393, -0.34893354773521423, 1.204884648323059, -1.143868327140808, 0.4905110001564026, -0.9972974061965942, 0.2492755800485611, 0.8328572511672974, 1.0767260789871216, -1.151818871498108, 0.4166393280029297, 0.09555280208587646, 1.402963399887085, 0.3338741660118103, -2.378502368927002, 1.2362303733825684, 0.7003299593925476, 0.19292502105236053, -0.056041326373815536, 0.16907168924808502, -0.56770920753479, -1.0123486518859863, 0.3456563651561737, -0.3003554940223694, -0.6252076625823975, 0.6008721590042114, -0.8476753830909729, -0.25566932559013367, -0.2949274182319641, 1.2382862567901611, 0.20269733667373657, -0.07283852249383926, 0.701026976108551, -0.49490347504615784, -1.9698309898376465, -0.46999314427375793, -0.18184205889701843, -1.3595635890960693, 0.5579313039779663, 0.7192986607551575, -0.37854665517807007, -0.933048665523529, 0.6147521734237671, 0.7705161571502686, -0.947089672088623, 0.5776655077934265, 0.5926478505134583, 1.1320033073425293, 0.7757192850112915, -0.4408031404018402, -0.4231683313846588, 0.04221402481198311, 0.3741784989833832, 0.38881754875183105, -0.7525570392608643, 1.4706588983535767, -0.4882391393184662, 0.8607191443443298, 0.1301112323999405, -1.3042958974838257, -0.5862047076225281, 0.33932170271873474, 0.3385559022426605], [-0.13398973643779755, -0.640155553817749, 1.7998591661453247, -1.293237328529358, -1.3043603897094727, 0.32426169514656067, 1.2104986906051636, 0.6073974967002869, -0.8421857357025146, -0.5363501310348511, 0.5548150539398193, -0.8824484348297119, -0.2634337246417999, -1.5742542743682861, -0.5891746282577515, -0.577958345413208, -1.037896990776062, -0.3446315824985504, -0.616769015789032, -0.3681032955646515, -1.4803804159164429, -0.5964298844337463, -1.4488542079925537, -0.11087022721767426, -0.2901117205619812, 0.431017130613327, -0.9732162356376648, 0.0014271429972723126, -0.00564511539414525, -1.4349414110183716, 0.5098928213119507, 0.3987478017807007, -0.4638347327709198, -0.44069844484329224, 0.5763387680053711, 0.3467826545238495, 0.8162006139755249, -1.0058709383010864, 0.9920852184295654, 0.9150201678276062, -1.6507729291915894, -0.21343903243541718, 0.7569667100906372, 0.01868770644068718, 0.5120618939399719, -1.1329188346862793, -0.08241133391857147, 0.3617171347141266, -0.9630151987075806, -0.5764431357383728, 0.4404814541339874, -0.8092737793922424, -1.157829761505127, 0.27123522758483887, -0.2675967812538147, 0.8331036567687988, -0.4204481244087219, 0.2472071647644043, -1.3565466403961182, 0.5562812089920044, -0.19865259528160095, -0.010018251836299896, -1.2215982675552368, 1.2351117134094238], [-0.5267093181610107, -0.7078009247779846, 1.0423554182052612, -1.2256994247436523, -0.9368893504142761, -0.14663659036159515, 0.6964132189750671, 0.5707158446311951, -1.282088041305542, 0.08100827038288116, 0.9598376750946045, 1.011986494064331, -1.0966252088546753, -0.18637719750404358, -0.2549550235271454, -0.5524161458015442, -0.3386031985282898, -0.2779443562030792, -1.889654517173767, 0.9466730356216431, 0.6699289679527283, -0.524344265460968, -1.1710209846496582, 1.0649805068969727, 0.3192351162433624, 2.932912826538086, 0.5224400758743286, -2.0618951320648193, -0.0009244779357686639, -1.2588226795196533, -1.429254174232483, 0.2111915647983551, -0.24404536187648773, 0.16839946806430817, 0.44719773530960083, 0.856335461139679, -0.27838051319122314, -0.1914598047733307, 2.028204917907715, -1.8336440324783325, 1.507688283920288, 0.06772259622812271, -0.13198064267635345, 0.8505218029022217, 1.0350509881973267, -0.248270183801651, -0.9826463460922241, 0.6945169568061829, -0.4171833395957947, -1.0608994960784912, -0.8812442421913147, -1.570694088935852, -1.2377923727035522, 0.5202172994613647, -1.073571801185608, 1.5851249694824219, 0.04243814945220947, -0.5040790438652039, -0.08470635861158371, -1.0330778360366821, 1.694214940071106, 0.4869946539402008, 2.978924036026001, 0.7115088701248169], [0.05084021016955376, -2.272132158279419, 1.1913199424743652, 0.3705199360847473, -0.42473122477531433, -0.11879249662160873, -0.5779892802238464, 0.9936654567718506, 0.7599894404411316, -0.02749255858361721, 1.8995540142059326, -0.5835047364234924, -0.01983977109193802, -0.7393052577972412, -0.6746383309364319, 0.011899692937731743, -1.889062762260437, -1.8522343635559082, 2.189423084259033, -1.1967439651489258, 1.0925650596618652, -1.3555043935775757, 1.0032155513763428, 1.556401252746582, 3.2633142471313477, -2.486530303955078, -1.3849937915802002, 1.159729242324829, 0.7257177829742432, -1.693372368812561, 0.46977269649505615, -0.8644357323646545, -0.8857221007347107, -0.750678300857544, -0.874980092048645, 1.4639207124710083, 0.02739631198346615, -1.2777475118637085, -0.23971156775951385, -0.6478890180587769, 0.48101291060447693, 2.713981866836548, -0.1824345588684082, -1.899751901626587, -0.1322954297065735, -0.1680113524198532, 0.41789403557777405, -1.4001643657684326, -0.7957239747047424, -0.11329580843448639, -1.4424644708633423, -1.2004499435424805, -1.6053621768951416, 0.5822890996932983, 0.7384276390075684, 0.7353358268737793, -0.7409637570381165, -0.5963114500045776, 1.079367995262146, -0.2022385597229004, 0.24239540100097656, 0.1260100156068802, -0.35372394323349, -0.6941772103309631], [0.9532274603843689, 1.2420779466629028, 0.4137590527534485, 0.11531054228544235, -0.5778430700302124, -0.30837443470954895, -1.0434520244598389, 0.4900355041027069, 0.13951964676380157, 0.5168839693069458, -1.3279308080673218, 2.8679018020629883, 0.15247981250286102, -0.3568875789642334, -1.3763766288757324, -0.8816028833389282, 0.6886472105979919, 0.9311915040016174, 1.905781626701355, -1.0373287200927734, -0.610984742641449, -1.9536832571029663, -0.03771331161260605, 0.744036078453064, 2.0003092288970947, 1.5813497304916382, 0.6873915791511536, -0.15233156085014343, 0.5810502767562866, 0.8293691873550415, -0.9568465948104858, 1.6999030113220215, -0.7981995344161987, 0.9539733529090881, -1.5402547121047974, 0.11732490360736847, -0.12875865399837494, 0.40582743287086487, 1.080601453781128, -0.8009663820266724, 0.10673577338457108, 0.9833114147186279, -0.41684120893478394, 0.631583034992218, 0.1909143626689911, 2.391181707382202, 0.9483626484870911, 1.2692878246307373, -1.435640811920166, 0.5141074657440186, 0.8445183038711548, 1.359061598777771, 1.6400200128555298, -0.890222430229187, -0.10685694962739944, -1.0480996370315552, -0.16808128356933594, 0.7719356417655945, -0.3689377009868622, 0.049838680773973465, 0.539313793182373, -0.6375845074653625, -1.638596773147583, -0.4272097051143646], [0.9171552062034607, 0.857119083404541, -1.304264783859253, -0.24115271866321564, 1.5644505023956299, 0.8023303151130676, -0.6129562854766846, 1.3856042623519897, 0.29540401697158813, -0.19185870885849, -0.18503950536251068, -0.10298753529787064, 1.2846970558166504, -0.2761589586734772, -0.8017401099205017, 0.15132665634155273, 0.15736563503742218, 1.4424694776535034, 0.8138443231582642, -0.29045289754867554, -1.3969260454177856, 1.4543907642364502, 0.3929669260978699, 0.5553497076034546, 0.24555636942386627, -0.1278192698955536, -0.024278860539197922, 2.4757039546966553, 1.8093069791793823, 0.08187524229288101, 0.2629242241382599, -1.0765303373336792, 1.310573697090149, 1.3621747493743896, -0.0572977289557457, -2.1062209606170654, 2.3098888397216797, -0.16319186985492706, -0.6011064052581787, -0.8572262525558472, 0.4207751154899597, -1.102827548980713, 0.3822548985481262, 0.9153040051460266, 1.1186895370483398, -0.9251834154129028, 0.19065062701702118, 0.5353222489356995, 0.2545355260372162, -0.2032010555267334, -0.9796038866043091, -0.5457821488380432, 0.9838692545890808, 0.9398505687713623, -0.9641216993331909, -0.08544593304395676, -0.5657476782798767, 1.4193490743637085, -0.9623637795448303, 1.591849446296692, 0.5846052169799805, -1.229514479637146, -0.707206666469574, -0.19419220089912415], [-0.11173807084560394, 1.920534372329712, 0.2891436219215393, -0.26041796803474426, -0.05655699595808983, -1.054498553276062, -0.1574007272720337, 0.9150384664535522, -0.15644364058971405, 1.1315900087356567, -0.11114402860403061, -0.3826734125614166, -2.242567539215088, -1.5342994928359985, -0.23245121538639069, -1.3690372705459595, -0.36024144291877747, 0.2842368483543396, -1.0827070474624634, 0.19166287779808044, -0.08590836822986603, 0.16071811318397522, -1.529322624206543, -1.1165393590927124, -0.57552570104599, -0.6443792581558228, -0.7468743920326233, -2.4556190967559814, 1.8374167680740356, 0.420602411031723, 1.933780312538147, 0.8071696758270264, -1.4668347835540771, -0.015858709812164307, -1.0628554821014404, 0.4456974267959595, -1.2364808320999146, -0.6318315267562866, 0.7444415092468262, -0.26767969131469727, 0.31129324436187744, 1.2699673175811768, -1.1798003911972046, 0.3462001085281372, 0.16920509934425354, 0.6797260642051697, -1.4648903608322144, -0.025113599374890327, -0.23538561165332794, 0.3768364191055298, 1.5506131649017334, -0.6507202982902527, -3.0014162063598633, -0.9893770813941956, -0.9474010467529297, 0.7828783392906189, 0.7488701939582825, -0.7908009886741638, 0.15852950513362885, -0.16736850142478943, -0.3672586977481842, 1.5489165782928467, -0.9818601012229919, 0.6349430680274963], [0.3961876332759857, 0.6317539215087891, -1.1299636363983154, -1.0146865844726562, 0.2574392855167389, 0.29820457100868225, -0.4148772954940796, -1.000431776046753, -1.003609538078308, -0.9055464267730713, -1.5484644174575806, 0.4681127965450287, 0.27507123351097107, 0.5732461214065552, -0.4075060784816742, 0.3656895160675049, -0.9629161357879639, -1.2734744548797607, 1.2254087924957275, -1.012712001800537, 0.4975428283214569, 0.8093356490135193, 0.5505187511444092, -0.22111521661281586, 0.72984778881073, 1.5254881381988525, 1.7853734493255615, 0.2831525504589081, 0.27369362115859985, 1.3566128015518188, -0.7355672717094421, -0.12412422895431519, -1.6273462772369385, 2.7442469596862793, 1.948844075202942, 1.700771450996399, -0.17314735054969788, 0.3378615379333496, -0.9244936108589172, -1.1905944347381592, -0.19960643351078033, 1.3969155550003052, 1.6423637866973877, -0.822717010974884, 0.29857146739959717, 0.7034040093421936, 0.2592802047729492, -0.013944385573267937, 0.263422429561615, -1.2312486171722412, -0.3403494358062744, -1.2422749996185303, 1.556087613105774, -0.9270965456962585, 0.529725968837738, -0.044798530638217926, 0.6442126631736755, -0.2841833531856537, 1.007509708404541, -1.2294087409973145, 2.1927242279052734, -0.6449803113937378, -0.9337954521179199, 1.9301596879959106], [0.34057438373565674, -1.0680571794509888, -1.4066787958145142, 0.9921483397483826, 0.06492801010608673, 0.5540438294410706, -0.1227274164557457, -2.6226348876953125, -0.6653223633766174, 3.1865241527557373, 0.9210699200630188, -0.06308238953351974, 1.2699024677276611, -0.7941576838493347, 1.0553481578826904, -0.07365227490663528, -1.5284144878387451, 1.4181288480758667, 1.3450835943222046, 0.814472496509552, 0.29541653394699097, -0.023893307894468307, -1.2396141290664673, -0.14732077717781067, -0.26122066378593445, -0.8158160448074341, -0.5682453513145447, -0.5391733646392822, 1.3254607915878296, -0.8255376815795898, 2.1024441719055176, -0.7251946926116943, -1.442917823791504, -1.4177417755126953, -1.049961805343628, 1.1324453353881836, 0.3268444240093231, -1.40959632396698, 0.23005852103233337, -0.7644059062004089, 1.154296636581421, -1.4270234107971191, -1.648206353187561, 0.6129335165023804, -0.35100412368774414, -0.5470098853111267, 1.3070199489593506, -0.7933192849159241, 0.2632029950618744, 0.9099840521812439, -0.7418670058250427, 0.9059901237487793, 0.09708821028470993, 0.7097779512405396, -0.6881433725357056, -1.5524308681488037, -1.0806175470352173, -0.2855944335460663, -0.6281599998474121, 0.9769405722618103, 0.35388144850730896, -1.5773569345474243, 1.4899674654006958, 0.06281019002199173], [-1.5680954456329346, -1.2261666059494019, 0.6501659750938416, 0.3790178894996643, 0.3574136197566986, 0.5076532959938049, 0.16028328239917755, 1.517425537109375, -0.0686485543847084, -0.023148665204644203, 0.7060842514038086, 0.23504917323589325, 0.08268625289201736, 1.094160556793213, 0.07485257834196091, 2.8625612258911133, 0.17661653459072113, -1.0809283256530762, 0.37353330850601196, 1.8475638628005981, 0.47254762053489685, 0.5782954096794128, 0.7702692747116089, 1.1866363286972046, -0.9839362502098083, 0.24227258563041687, -0.25748637318611145, -0.09266330301761627, 0.039270609617233276, -0.43913090229034424, 0.07583299279212952, 2.3076658248901367, 0.30992212891578674, -0.417436420917511, -0.3080368936061859, 0.25313448905944824, 1.4643727540969849, 0.7240367531776428, 1.075669527053833, -0.6105254292488098, 0.5629537105560303, 0.9672962427139282, 0.4117412865161896, 1.4687000513076782, 0.8597769737243652, 0.7952724695205688, 0.8458490967750549, 0.22027097642421722, -1.085115909576416, -1.4595922231674194, -0.1494416445493698, -2.1935617923736572, 0.7392070293426514, 0.3072322905063629, -0.016757048666477203, 0.41605105996131897, -2.543989419937134, -0.33676642179489136, 0.7422778606414795, 1.6098568439483643, -0.03378758952021599, 1.186204195022583, -0.2384095937013626, 0.4518202841281891], [0.17526555061340332, -0.9131906628608704, -1.1507596969604492, -0.9211735725402832, -1.113899827003479, -0.9730199575424194, -0.04590662568807602, -0.14775346219539642, -0.2894214391708374, -0.2660068869590759, -0.7576815485954285, -0.10269795358181, 0.7543103098869324, -0.030789947137236595, -0.7736948728561401, -1.246199607849121, 1.0027306079864502, 1.7858071327209473, 0.5671119689941406, 0.8664771318435669, -0.021496566012501717, -1.0470243692398071, 0.5916492938995361, 1.3104033470153809, -0.8690069317817688, -0.47449207305908203, 0.03030744381248951, 0.20635128021240234, -1.4117769002914429, 0.11185789853334427, 0.3996575176715851, 0.13788817822933197, 1.9301912784576416, -0.22143688797950745, -0.504613995552063, 0.40985313057899475, -0.7572275996208191, -0.7678825855255127, 2.2765450477600098, -0.1809074431657791, -0.821127712726593, 2.098068952560425, -0.11210059374570847, -1.8735378980636597, -0.4633651375770569, 3.7545905113220215, 2.160597085952759, -1.7740923166275024, -0.8536866903305054, 1.1812572479248047, -1.3624228239059448, 0.5334473848342896, 0.7041444778442383, 0.7369796633720398, 0.014166994951665401, 0.26527276635169983, -0.8193018436431885, -0.05272471532225609, -0.5682211518287659, -0.8971980214118958, 1.6928002834320068, 0.7226894497871399, 1.631662130355835, 1.6240724325180054], [0.09453582018613815, -2.0015783309936523, -1.4580647945404053, 1.0092008113861084, -0.04460703209042549, 0.16180244088172913, 0.9288837909698486, -1.5436947345733643, -0.3263958692550659, 0.04836186021566391, -0.48350808024406433, -0.27063482999801636, 1.0248243808746338, 1.12433922290802, -0.474765807390213, 0.7016538977622986, 1.1117533445358276, -1.546800971031189, -0.8759575486183167, 0.6025012731552124, 1.1327269077301025, 0.48835793137550354, 0.3604067265987396, 1.107466220855713, 0.4063476324081421, 2.197256565093994, -1.3242396116256714, 1.8719642162322998, -0.08756859600543976, 1.2459514141082764, 0.13744154572486877, 0.9788733720779419, 0.0957435742020607, 0.2357671558856964, -0.874112069606781, -0.019480040296912193, 1.2530076503753662, 0.7869395613670349, -0.44475626945495605, 0.3020293712615967, -2.626277208328247, -0.0659426897764206, -1.5158847570419312, 1.6775317192077637, 1.8277441263198853, 1.8439974784851074, 0.08650172501802444, 0.058672841638326645, -0.13942615687847137, -0.2038014680147171, 1.1438473463058472, 0.5782312750816345, -1.2836006879806519, -1.4371706247329712, 1.3428761959075928, -0.501899778842926, -0.8623862862586975, -0.8156372904777527, -0.6811889410018921, -0.14893853664398193, 0.5835290551185608, -0.4784509539604187, 0.004154330585151911, 1.0029528141021729], [0.9853805303573608, 2.3028481006622314, -0.22907902300357819, 0.3524128496646881, -0.37886038422584534, 0.0686851367354393, -0.255321204662323, 1.9027236700057983, -0.3329075276851654, -0.154422789812088, -0.28601589798927307, -0.23582467436790466, 0.1581113636493683, 0.6192513108253479, -0.1202441155910492, -1.658586859703064, 2.3899285793304443, 0.18905510008335114, -0.5654679536819458, -0.5235117077827454, -0.6350369453430176, -1.7099064588546753, -1.1478520631790161, 0.9610949158668518, 1.0536620616912842, -0.04760909453034401, 0.24104636907577515, 0.8737420439720154, -0.21597547829151154, 0.7514007687568665, 0.18923114240169525, -0.877907931804657, -0.12755505740642548, 1.354197382926941, 0.019319219514727592, 1.4661213159561157, 0.39280903339385986, 0.005286120809614658, -1.4581667184829712, -0.2510814070701599, 0.7546224594116211, -0.43398407101631165, 0.770952045917511, -0.5603587627410889, 0.6022900342941284, 1.096903920173645, -0.029735147953033447, 0.35312384366989136, -0.6811121106147766, -0.09277995675802231, 1.7045575380325317, -0.3708450496196747, -0.9082017540931702, -0.28347304463386536, 1.032886266708374, -0.41911762952804565, -1.4714337587356567, -0.08066203445196152, -0.6032735109329224, 0.013695310801267624, -0.41622576117515564, -0.5863186120986938, -0.03450166806578636, -1.16361665725708], [0.24499407410621643, -1.4416615962982178, 2.441138982772827, 0.6952856779098511, -0.665790855884552, -0.2313290536403656, -1.038956880569458, 0.40328487753868103, 0.5965243577957153, 1.2398093938827515, 1.2558318376541138, -0.19938179850578308, 0.09086696058511734, -0.20301030576229095, 0.2201012521982193, 1.0433019399642944, -0.005017926450818777, -0.5195494294166565, 0.14664378762245178, -0.43045341968536377, -1.0355654954910278, -0.5716727375984192, -0.9444006681442261, -0.054481979459524155, 0.02933034859597683, -1.8755855560302734, 0.12529303133487701, 0.7069602608680725, 0.32000914216041565, -1.5595892667770386, -0.7192103266716003, -1.1856577396392822, 0.6024808883666992, 1.9492360353469849, 0.5517402291297913, 2.598172426223755, 0.09906533360481262, 0.4329985976219177, -0.5717892646789551, -0.04445084184408188, -2.4204249382019043, 0.6334881782531738, 1.2905604839324951, -2.730767250061035, 0.35811513662338257, -0.040823452174663544, 0.028773417696356773, -0.6623984575271606, 0.26030778884887695, -0.35217803716659546, -0.5028626322746277, 0.16236691176891327, -0.9347649812698364, 0.3683534860610962, 2.474059581756592, -1.418160319328308, -0.4155764877796173, -1.5645827054977417, -1.386925220489502, 0.034352611750364304, 0.12651661038398743, 0.17549721896648407, 0.018348002806305885, -0.5916258096694946], [0.9093884229660034, 0.4126776158809662, 0.3332895040512085, -0.7555898427963257, -0.4847029447555542, -0.5464232563972473, 0.6812087297439575, 1.0226434469223022, 1.287951111793518, -1.7345706224441528, -1.2039257287979126, 0.40165889263153076, 0.028043588623404503, 1.305286169052124, 0.7991899847984314, -0.4390918016433716, 0.5417144894599915, -0.674598217010498, 0.32737085223197937, -0.7548435926437378, -0.9082781076431274, 1.2784596681594849, 1.3079532384872437, -1.8244763612747192, 2.880695343017578, 0.4161255359649658, 1.1947546005249023, 0.17613963782787323, -0.16710014641284943, 1.5558176040649414, 0.5126445889472961, 0.42019158601760864, 1.602968692779541, -0.7242704033851624, 0.1650054156780243, 0.626230776309967, -0.10971743613481522, -0.5193496942520142, 1.509589672088623, 1.4445652961730957, -1.5091502666473389, 0.3216334283351898, -0.207960307598114, 1.010543704032898, 0.8128827810287476, -0.5454908609390259, -0.6391041874885559, -0.5728989839553833, -0.2728680372238159, 0.8010709285736084, -0.17867949604988098, -0.9892365336418152, 0.7175029516220093, 1.053977370262146, 1.2046884298324585, -0.06242114305496216, 0.8066504597663879, -0.4946655035018921, 1.5189236402511597, -1.4016361236572266, 0.9742858409881592, -0.48795634508132935, -0.4202723801136017, -0.4620126485824585]], [[0.06185400113463402, -0.8235948085784912, 0.19253064692020416, 0.2827695608139038, -1.3714927434921265, 0.5311341881752014, 0.86271071434021, 1.5629594326019287, -0.10588192939758301, 0.46681615710258484, -0.16774162650108337, -1.0486611127853394, 1.7207000255584717, -1.1576908826828003, 1.1262727975845337, 1.0041074752807617, 0.5838934183120728, -1.2609498500823975, -1.5902308225631714, -0.7676109671592712, -0.00577115174382925, 1.8010281324386597, -0.6872944831848145, 0.6379814743995667, -1.8714624643325806, 0.48010149598121643, 0.7186427712440491, -0.9268929362297058, 0.15836840867996216, 1.5475445985794067, -0.7555047273635864, 1.011986494064331, 0.6779957413673401, -1.5188812017440796, 0.2688027322292328, -0.7787172794342041, -0.04914805665612221, -0.2758077383041382, -0.3099488914012909, 0.17023533582687378, -0.9076672792434692, -0.98777174949646, -0.6719154119491577, 1.302855372428894, 0.26654958724975586, -1.03343665599823, -0.3674343228340149, 0.4713963270187378, -0.34701162576675415, -0.14605388045310974, 1.4909985065460205, 0.16891898214817047, -0.8413754105567932, -1.1552656888961792, 0.23974063992500305, -0.2868299186229706, -0.558385968208313, -0.9529391527175903, 2.755465030670166, -0.47368136048316956, -2.6118218898773193, 0.5705859661102295, -0.4540863633155823, -0.5117237567901611], [1.2610797882080078, 0.1622021645307541, 1.0659250020980835, -0.6257295608520508, 0.9943594336509705, -1.929425597190857, -1.1747099161148071, -0.5001630187034607, -1.2368146181106567, -0.3299981653690338, 0.4016243517398834, 0.39823925495147705, -0.9488258361816406, 0.5657315254211426, 0.9188989400863647, 0.06100428104400635, -0.8706819415092468, -1.0866105556488037, 0.5615800619125366, -0.43749287724494934, -1.0703085660934448, -0.7618817687034607, -0.953783392906189, 1.8416588306427002, -0.7265748977661133, 0.3985813856124878, 0.5797430276870728, 0.1289759874343872, -0.6001328825950623, 0.19157500565052032, -1.1096150875091553, -0.5099155902862549, 0.0015042823506519198, 0.6982952356338501, 0.29394665360450745, -0.4076201617717743, -0.06950968503952026, 0.07254332304000854, -0.5040217638015747, 0.1327599734067917, -1.045316219329834, -0.5656304359436035, -0.15512505173683167, 1.123058557510376, -1.097690463066101, 0.8297274112701416, -0.30077478289604187, -0.3215172290802002, 0.7626104950904846, 0.09895892441272736, 1.0377404689788818, -0.9279342889785767, -0.8638995885848999, -0.3008611500263214, -0.12544666230678558, 1.042510747909546, 1.3948190212249756, 2.3855342864990234, 0.09519315510988235, -0.28648465871810913, -1.7325999736785889, -0.15880808234214783, -0.02989904396235943, 1.606360912322998], [0.6196789145469666, 0.893474280834198, 0.5350542068481445, -0.8954226970672607, -1.2786463499069214, 0.2305578887462616, -0.5466042757034302, -1.3636795282363892, -0.0836743712425232, 0.5219612121582031, 0.6526864171028137, -1.275860071182251, 1.5139663219451904, -0.35137584805488586, 1.5727086067199707, -0.5589144825935364, 0.2851024866104126, -0.5638794898986816, 0.5935974717140198, -1.2036594152450562, 0.004781309049576521, 0.971307635307312, -0.09043430536985397, 0.8708606362342834, 0.1941695362329483, 1.2696675062179565, 1.5186153650283813, -0.8863188028335571, 0.0422031544148922, -1.1952134370803833, -0.13879798352718353, -0.3746224641799927, -0.2961420714855194, -1.0876222848892212, 2.1036970615386963, 0.16872933506965637, -1.2715858221054077, 0.17257677018642426, -0.0600522942841053, 0.7457751035690308, 2.3427605628967285, 1.895795464515686, -0.8921986222267151, 0.42326319217681885, 0.10647274553775787, 0.5399969220161438, -0.7632986307144165, -1.1476078033447266, 0.8240695595741272, 1.1224756240844727, -0.13144780695438385, 1.2686514854431152, -1.4415686130523682, -0.29990166425704956, 0.70405513048172, 0.24875588715076447, -0.37554094195365906, -0.16674432158470154, 0.8455114960670471, -2.0363996028900146, 1.1411443948745728, -0.9132367968559265, 0.17793340981006622, 0.6634895205497742], [0.29133671522140503, 0.34819456934928894, -0.8149849772453308, -1.340802550315857, 1.7122318744659424, 0.17274466156959534, 0.787248432636261, 0.8827417492866516, 2.1594862937927246, -1.2643367052078247, 2.4786338806152344, 0.4989679157733917, -1.4617087841033936, -1.1927217245101929, -0.7678903937339783, -0.9833356738090515, 0.13090503215789795, 0.12597812712192535, -0.9208294749259949, 0.128226637840271, 2.1825098991394043, -0.9891469478607178, 0.41121581196784973, -0.8094167113304138, 0.7086142897605896, 0.0007269051275216043, 1.561058521270752, -1.0608186721801758, -1.2098277807235718, -0.23781119287014008, 0.07851480692625046, 0.6444214582443237, 0.4054326117038727, -1.015671730041504, -0.9476436376571655, -0.9416036605834961, 0.8869562149047852, -0.8112519383430481, -0.052100662142038345, 1.0570225715637207, 0.4010462462902069, -0.33372822403907776, 0.6219961047172546, 0.7951924204826355, -0.37440183758735657, -0.4406386911869049, -1.7185693979263306, 0.7341101765632629, 1.0722265243530273, 0.44185805320739746, 0.7343413233757019, 0.2864750027656555, -0.7877167463302612, -0.01321711391210556, 2.097642183303833, 0.7285893559455872, -0.6734583377838135, -0.09775689244270325, 0.21009992063045502, 0.5349825620651245, 1.5255190134048462, -0.6564925312995911, 0.5944488644599915, -0.10979504883289337], [-0.2563685476779938, -1.1611230373382568, -0.5334506630897522, -0.8268241286277771, 0.07706084847450256, 0.9245995283126831, -1.79026198387146, 0.726878821849823, 0.947252631187439, 0.4765002429485321, 0.9189902544021606, 1.0294126272201538, -0.42775759100914, 0.06108104810118675, -0.6519219875335693, 1.3057018518447876, 0.47094979882240295, -0.7177333831787109, -1.1499528884887695, 0.5728737115859985, 1.627364158630371, 0.5052794814109802, -0.22764882445335388, 0.5493429899215698, 1.7045947313308716, 0.6696248054504395, 0.8022226095199585, -0.2839326858520508, -0.1284743994474411, 0.3435373604297638, -0.6634503602981567, 0.4942722022533417, -0.7834247350692749, 2.309488296508789, 0.9379700422286987, 0.09456387162208557, 0.8286904096603394, 0.42229869961738586, 1.264223337173462, -0.49760064482688904, -0.21020589768886566, -0.7678739428520203, 0.7171269059181213, -0.4624868333339691, 0.10254267603158951, 0.7649878263473511, -0.8542622923851013, 2.581150531768799, 0.20574074983596802, -1.6334881782531738, 1.3368782997131348, -2.0698134899139404, -0.8324936032295227, -1.4497652053833008, 1.547279953956604, -0.7721930146217346, -0.7415318489074707, 0.47476401925086975, 0.14129725098609924, -0.498922199010849, -1.396567463874817, -1.362375259399414, 2.127784490585327, 0.7028341889381409], [1.574469804763794, -0.5705477595329285, 0.7261847257614136, 0.5441359877586365, 1.5105489492416382, 0.4052809178829193, 0.18724998831748962, 0.45098644495010376, -1.3905514478683472, -0.05148174986243248, -0.27509579062461853, 2.0225038528442383, -0.40389740467071533, 1.2886384725570679, 1.7476253509521484, -0.32737040519714355, -0.9801255464553833, -0.9850638508796692, 0.48044857382774353, -0.6371172070503235, -0.01014840416610241, 0.3733310103416443, 0.08426021784543991, -1.558786153793335, -1.8515253067016602, -0.200914666056633, -0.09325302392244339, -0.6511587500572205, 1.2411651611328125, 1.0139038562774658, 0.03858187794685364, -0.2254752218723297, 0.705336332321167, 0.081965371966362, -1.092596173286438, 1.1023430824279785, -0.43263038992881775, -1.1282604932785034, -0.7154102921485901, -1.4215904474258423, 0.37978997826576233, 0.478644996881485, -1.8022581338882446, 1.7526981830596924, 0.5375781059265137, 0.5240506529808044, -2.7090396881103516, 2.123511552810669, 2.4499850273132324, 0.8212078809738159, 0.02878270298242569, -0.8028417229652405, -0.8852238059043884, -1.5470821857452393, -0.8367324471473694, 0.7446463108062744, 0.6111398339271545, 0.6916578412055969, 2.199479818344116, 0.3045819401741028, -0.9473174810409546, 1.06771981716156, 0.6586836576461792, 0.41322755813598633], [-0.25975438952445984, 1.2012661695480347, 0.3859541118144989, 1.513479232788086, 0.41686591506004333, -1.05305814743042, -0.3758375346660614, 1.3912492990493774, -0.42736804485321045, -0.44542595744132996, 1.340572476387024, -0.008340488187968731, -1.1319077014923096, -1.161372184753418, 0.03538142517209053, 0.5253288745880127, 0.41251465678215027, -0.2879208028316498, -1.7062922716140747, 1.1490094661712646, 0.2512683868408203, -0.18504273891448975, 0.41231584548950195, -1.453011393547058, -1.148270845413208, -0.06146597862243652, -0.6165775656700134, 0.6655429005622864, -0.23462098836898804, -0.10184692591428757, 0.31414246559143066, 0.22472336888313293, 0.057330742478370667, -0.2911575734615326, -0.28559064865112305, 0.4921071231365204, -0.6334257125854492, -1.132347583770752, -0.8758174777030945, 0.9589250087738037, -0.4263562560081482, 0.16420283913612366, 0.14000436663627625, 1.794144868850708, -0.7939204573631287, 1.1331992149353027, -0.8725574016571045, 0.6220085620880127, 1.6719824075698853, 1.096543788909912, 0.18043839931488037, 1.7548339366912842, -1.0919972658157349, -0.023213285952806473, -0.42239251732826233, 1.3486319780349731, 0.029535464942455292, 0.0012617438333109021, 0.7449546456336975, -1.0843534469604492, 2.0064682960510254, -0.886530876159668, -0.15870226919651031, -0.04305250570178032], [0.6507193446159363, 0.9048263430595398, -0.2806227505207062, -0.7466864585876465, 1.1647893190383911, 1.3172268867492676, 0.08914587646722794, -2.698672294616699, 1.2572643756866455, 1.7551847696304321, -1.5809667110443115, -0.6091871857643127, 0.9597101211547852, 0.7224075198173523, -0.04991799220442772, -1.8484777212142944, 0.33211788535118103, 0.6157134771347046, -0.9150558114051819, 0.037223391234874725, -0.11154726147651672, 0.2993164360523224, -0.7086700201034546, 1.9281985759735107, -1.37445068359375, -0.2663891613483429, 0.8613665103912354, 0.37095412611961365, -0.5357948541641235, -1.0719401836395264, -0.8806555271148682, 0.5635249614715576, 0.13599660992622375, -0.9358574151992798, -2.054690361022949, 0.47674524784088135, -0.050274722278118134, 0.8654171228408813, -2.393645763397217, 0.6167794466018677, -0.7046111226081848, 0.38958126306533813, -0.6504655480384827, 0.06498895585536957, -1.7493932247161865, 0.9211990237236023, 1.0037485361099243, -0.32412609457969666, -1.958371877670288, 1.33308744430542, 1.1328750848770142, 0.2587408125400543, 0.9884172081947327, -0.9687871336936951, 1.9005359411239624, 0.00985549483448267, 2.86179256439209, -0.37352126836776733, 0.5938159227371216, -0.2692272663116455, 0.2084045559167862, -0.8016931414604187, -0.09959963709115982, 0.5138639211654663], [0.40122073888778687, 0.9293825626373291, -1.1313037872314453, 1.1331714391708374, -2.111051559448242, -0.3604864776134491, 0.19846731424331665, -1.2077959775924683, -2.9180386066436768, -0.1568303257226944, -0.4671393930912018, 0.6604037880897522, 0.6489934921264648, -1.5652586221694946, 0.3425333797931671, -0.29151812195777893, -0.6398760676383972, 0.9977406859397888, -0.5202412605285645, 0.8652810454368591, -0.1749047040939331, -1.652940034866333, -0.09210321307182312, -1.1974760293960571, -0.8740143179893494, -1.9622143507003784, -0.5478892922401428, 1.0735944509506226, 0.5700066685676575, 0.18349769711494446, -0.5498981475830078, -1.535059928894043, -0.24759095907211304, 0.2743837237358093, -0.42601075768470764, -0.9602908492088318, 0.8861318826675415, 0.9400697946548462, 0.8325326442718506, 0.45431479811668396, 0.22062894701957703, 0.7449164390563965, -2.6603989601135254, 0.22416619956493378, 0.8294802308082581, 0.05832940712571144, -0.9769405722618103, -0.1687294989824295, 2.977581024169922, 1.2764455080032349, -0.9302740693092346, 0.8511533141136169, 1.791015386581421, -0.8693352937698364, 0.2244851142168045, -1.338088870048523, -0.1390024721622467, 0.37080317735671997, 0.9631152153015137, -1.2267258167266846, -0.6673558354377747, 0.5948562026023865, 1.4404501914978027, 0.8025141954421997], [1.3209643363952637, -0.19575177133083344, 2.1661720275878906, -1.0548686981201172, -0.11883620917797089, 0.36616528034210205, -1.949117660522461, -1.9768917560577393, 1.7638139724731445, 0.3219478130340576, 0.9300961494445801, -1.2549619674682617, -1.586333155632019, -1.8212629556655884, -0.39690542221069336, -1.098192811012268, 0.21320921182632446, 1.2173513174057007, 0.25473475456237793, 0.48812001943588257, -2.0160281658172607, -1.4605426788330078, 2.207674503326416, 0.13514915108680725, -0.5561659336090088, -1.7760878801345825, 2.2848777770996094, -1.4444833993911743, -0.6913607716560364, -0.47760164737701416, -1.081875205039978, 0.4240390658378601, 0.8840951323509216, 1.0509251356124878, 0.03792191669344902, 0.09059120714664459, -2.1326584815979004, -1.2035515308380127, -1.1553223133087158, 0.35743486881256104, -1.1416237354278564, 0.6834484338760376, -0.414051353931427, -1.5006132125854492, 0.12748752534389496, 1.2491804361343384, 0.79909348487854, -1.3214118480682373, 1.1702206134796143, -0.5781773328781128, 0.6296591758728027, -0.2938775420188904, 0.6393259763717651, -1.3581620454788208, -0.1609801948070526, -0.40059757232666016, 1.2707133293151855, -0.9297778606414795, -0.3191225528717041, -1.2941062450408936, -0.004668351728469133, 0.40292730927467346, -1.7929682731628418, -1.1170262098312378], [0.7086396217346191, 0.5895422101020813, 0.9729563593864441, 1.4866887331008911, 0.06667663156986237, -1.036597490310669, 1.2006776332855225, 0.9113695621490479, -1.026606798171997, -1.0161269903182983, 0.4326428174972534, 0.11256548017263412, 1.270849347114563, 0.3767235577106476, 0.179023876786232, -0.7001223564147949, 1.9900202751159668, -0.0452120304107666, -0.19891691207885742, 0.18248294293880463, -0.28082019090652466, -0.31760942935943604, 0.01991775445640087, 0.12653526663780212, -0.32907578349113464, -0.14953064918518066, -0.9450263381004333, -0.28994014859199524, 2.397185802459717, 0.7111168503761292, 0.08698765933513641, 0.8357647657394409, 0.4684104919433594, -1.030336856842041, 0.5200878977775574, -1.1662989854812622, 2.1611204147338867, -0.0905655026435852, 0.31704989075660706, -0.682661771774292, -0.8438066244125366, 0.3298890292644501, 0.5399900674819946, -1.1233761310577393, 0.07468333095312119, -2.070181369781494, 0.21538420021533966, -0.65260249376297, -0.048578765243291855, 1.3060352802276611, 1.276898741722107, 1.0129481554031372, 0.12491258978843689, -0.3741259276866913, 1.2206134796142578, 0.3154789209365845, 1.2584428787231445, 1.2133889198303223, 0.48620882630348206, -0.9453436732292175, -1.041632890701294, 0.8175331354141235, -1.192070484161377, 0.6488847732543945], [-1.7131770849227905, -0.089823417365551, -1.2736390829086304, 0.25750732421875, -1.0384107828140259, 0.008930769748985767, -0.7440053820610046, -0.8403428792953491, -0.386893630027771, 0.10656683146953583, -0.8630138039588928, -0.6869087219238281, -0.007120821159332991, -0.9593808054924011, -1.0286685228347778, 0.38392820954322815, -0.45715856552124023, 0.46200111508369446, -0.42548444867134094, 0.9790414571762085, 1.5015062093734741, 0.5419349074363708, 0.0010868889512494206, -0.5904126763343811, 1.0364789962768555, -0.41312119364738464, 0.9307724833488464, -0.21967901289463043, 0.7096748352050781, -0.6499179601669312, -2.0347869396209717, 0.570990800857544, -0.472374826669693, 2.3030664920806885, -1.013085126876831, 0.7817960381507874, 1.6489850282669067, 0.46113118529319763, -0.2904636561870575, 1.8608592748641968, -0.26522064208984375, 1.3431025743484497, -3.09420108795166, 0.23057998716831207, -0.19447749853134155, -0.6668264865875244, -1.206852912902832, -1.0793139934539795, 1.5914313793182373, -0.04552832990884781, 0.08071563392877579, 0.7935624718666077, 0.6194254755973816, -1.4427953958511353, 1.286824107170105, -0.37935733795166016, 1.0995044708251953, -0.830790102481842, -0.8112942576408386, -0.06938832253217697, 1.3556413650512695, -0.29022103548049927, 0.7605552077293396, 0.3477635085582733], [-1.9339443445205688, -0.482179194688797, -1.9973177909851074, -0.2115863859653473, 0.004010844975709915, 0.051128800958395004, -0.7974879741668701, 2.467653512954712, 0.9780679941177368, 0.29941490292549133, -0.8047670722007751, -1.7800508737564087, -0.28828951716423035, -0.00903377402573824, 0.09980174154043198, -0.8228432536125183, -0.8071406483650208, -0.5349491834640503, 0.24340878427028656, -0.3384351432323456, 0.2056809663772583, -0.18007780611515045, -1.993038296699524, 0.958365261554718, -1.6903746128082275, -1.1313680410385132, 0.059515755623579025, -1.655645489692688, -0.5334650278091431, -1.0648343563079834, 0.7761645913124084, 0.7115360498428345, 1.5836633443832397, 0.15964558720588684, 0.20240439474582672, -0.4814683496952057, -1.3692926168441772, 0.8965765237808228, 0.3216054439544678, -0.32773253321647644, 0.27597111463546753, -0.6252961754798889, 1.1548993587493896, -1.6821794509887695, -1.1816375255584717, -1.046510934829712, -0.04964692145586014, -0.8591837286949158, 1.7346075773239136, -0.5405699014663696, -0.05212768167257309, -1.1737600564956665, -0.47222498059272766, -0.7293120622634888, -0.10817720741033554, -1.8881055116653442, -1.4024369716644287, 1.7109113931655884, -0.9056031703948975, -0.07238274812698364, 1.407344937324524, -0.32481372356414795, 0.855211079120636, 0.48898524045944214], [-0.3151620328426361, 0.5650559067726135, 1.5769292116165161, 0.4794299900531769, 0.4466516077518463, -0.5836386680603027, 0.42916926741600037, -0.020769864320755005, -1.067381501197815, -1.0822129249572754, -0.9937295913696289, 0.4257412850856781, -0.48852285742759705, -0.1529262214899063, -1.4030331373214722, 1.2461498975753784, -1.3626285791397095, 1.3974553346633911, 0.30319100618362427, -0.7391207814216614, 1.010229468345642, 0.6961972117424011, -0.28525489568710327, 0.4908912479877472, 1.8601562976837158, 0.028065018355846405, 0.3883231282234192, 0.16267095506191254, -0.38696110248565674, -0.09311620146036148, 0.9479560852050781, 0.1781514286994934, 0.1759747862815857, 0.9967004656791687, -0.34921014308929443, 1.2428789138793945, 0.7178913354873657, 0.396836519241333, 0.23967309296131134, -0.44156917929649353, -1.165794014930725, -0.3304893672466278, 0.2334575057029724, -0.7049907445907593, -0.9596838355064392, 0.4497007429599762, -0.5002169013023376, 0.7135840058326721, 0.4488650858402252, -0.5630092024803162, -0.25209230184555054, 0.7935229539871216, -0.8829476833343506, 0.6560383439064026, 0.11512833088636398, 0.23785026371479034, 1.0397570133209229, -0.46139073371887207, -2.1356301307678223, -0.09151952713727951, 0.4702100455760956, -0.8110668063163757, 2.96852970123291, 0.07379233837127686], [-0.29451385140419006, 0.1400361955165863, 2.1323745250701904, -0.3037891089916229, 1.4929429292678833, 0.7187389135360718, -1.822044014930725, -2.7635281085968018, 3.1088974475860596, -0.6714189648628235, 1.7742440700531006, 0.8401896953582764, 2.3783602714538574, -0.4722394645214081, -1.5549628734588623, -1.321537733078003, -1.1384730339050293, -0.40786921977996826, 1.3025394678115845, -2.2526652812957764, -1.134633183479309, 1.1615124940872192, -0.11567389965057373, 0.46498024463653564, -0.6986736059188843, -2.5476677417755127, 0.20710226893424988, -0.19290708005428314, 0.10545984655618668, 0.13269317150115967, 0.2685379087924957, 0.01942451111972332, 0.3539404571056366, -1.0807663202285767, 1.8976088762283325, 1.107832670211792, -0.4740369915962219, -1.1043702363967896, 1.1123465299606323, 1.0686720609664917, -0.6592000722885132, -0.9556788206100464, 0.48190316557884216, -2.1535120010375977, -0.1491282731294632, -0.49590781331062317, 1.730772614479065, -0.9625909328460693, 0.34160149097442627, -0.8077349662780762, -2.8047616481781006, 1.8508226871490479, -1.348455786705017, -1.0941418409347534, 1.039678931236267, 0.37689143419265747, -0.7922956943511963, -0.6039923429489136, -0.16897624731063843, -1.4182299375534058, -0.3471343517303467, -0.27951306104660034, 0.8000428080558777, 0.9767970442771912], [1.664932370185852, 1.0946400165557861, 0.7940214276313782, 0.21046574413776398, -0.5238152146339417, -0.770175576210022, 0.1818896234035492, 0.06491602212190628, -0.33107849955558777, -1.1957008838653564, 0.6030930876731873, -0.6318342685699463, 0.9338067770004272, -0.253030925989151, 1.7419860363006592, 0.20007847249507904, 0.5107613205909729, -0.9752915501594543, 0.46706080436706543, 0.5789914131164551, -0.013652723282575607, -0.5696702003479004, -0.7244613766670227, 1.7043848037719727, -1.513835072517395, -0.31158319115638733, -0.7230713963508606, 1.1941932439804077, 0.1068049967288971, -0.6810458302497864, -1.671629786491394, 0.7737864851951599, 1.2480006217956543, -0.30551043152809143, 0.48210588097572327, 0.40569907426834106, 0.7244998812675476, 0.1796293407678604, -1.5450764894485474, 0.09294870495796204, 0.07107627391815186, -0.18804262578487396, 0.6643715500831604, 1.1021080017089844, 1.0943408012390137, 1.0145988464355469, 0.20188483595848083, 0.6197437047958374, -0.48985204100608826, -0.5971758961677551, 0.793407678604126, 0.9824494123458862, -0.12041948735713959, -0.9171987175941467, 0.8243722915649414, -0.47911933064460754, 0.17388483881950378, -1.5686978101730347, 0.2119312584400177, -0.5990415811538696, -0.5564956068992615, 1.0077213048934937, 0.06068456918001175, 0.025167331099510193]], [[-2.2001209259033203, -0.1426621675491333, 0.020197194069623947, 0.3192386329174042, -0.16573050618171692, 0.14354798197746277, 0.48728305101394653, -0.3086296021938324, -1.0261517763137817, -0.3848457634449005, 1.8283039331436157, -1.0740768909454346, -0.8231782913208008, 1.4075512886047363, -1.3145859241485596, 0.28207725286483765, -0.6857382655143738, -0.2710734009742737, 0.14363351464271545, 0.5711228847503662, -0.5377380847930908, -1.600156545639038, 1.447962760925293, -0.7840142250061035, -0.014633748680353165, -0.7832173109054565, 0.49433019757270813, -0.8065822720527649, -0.935127317905426, -0.06134650856256485, 0.7127781510353088, 0.3801880180835724, -0.3817749321460724, 0.19079239666461945, 1.1219202280044556, 0.8697821497917175, -1.0397334098815918, 1.1908445358276367, 0.8479481339454651, 0.34873759746551514, 0.24308332800865173, -0.08370077610015869, 0.41668009757995605, -0.32194405794143677, -1.7117291688919067, 0.10659442842006683, -0.08503958582878113, -1.087611436843872, -0.043415311723947525, -0.3586272895336151, -0.1985875964164734, -0.9727531671524048, 0.876069962978363, 0.49104106426239014, 0.8324479460716248, -0.5226740837097168, 0.5771183967590332, 0.9130811095237732, -1.6782475709915161, 0.15919385850429535, 0.515839159488678, 1.0324840545654297, -0.2529771327972412, -0.2048211693763733], [0.30260559916496277, 0.34909769892692566, -0.5110004544258118, -1.8791571855545044, 1.4301621913909912, 1.2738665342330933, -0.19122380018234253, 1.6070188283920288, -2.104172468185425, -0.02902982197701931, 1.2189048528671265, -0.475911408662796, 1.009049654006958, -0.5831737518310547, 2.1954734325408936, 1.2904866933822632, 0.5581377744674683, -1.0374023914337158, -0.328338086605072, -0.7799020409584045, -0.7037825584411621, 0.0005521723651327193, 1.1118803024291992, -1.5933314561843872, 1.1048924922943115, -1.1856869459152222, -2.3288989067077637, 0.8117262721061707, 0.5990626215934753, -0.4146921932697296, -0.260614275932312, -0.6904717683792114, -0.14081920683383942, -0.8010792136192322, -0.4849558174610138, -0.642362117767334, 1.7079354524612427, 0.2852577567100525, -1.447368860244751, -0.17525383830070496, -1.1815091371536255, 0.22726836800575256, 0.295684278011322, 0.6938281059265137, -1.4073041677474976, 1.1122862100601196, 0.7581015825271606, 0.005540505982935429, 1.8129557371139526, -0.8256434202194214, 1.0499327182769775, 0.7917920351028442, -0.26509445905685425, -0.027607988566160202, -0.42344558238983154, -0.1753142923116684, 0.3506934642791748, 0.28156203031539917, 1.4291377067565918, -0.554897129535675, 0.6172391176223755, -0.08227526396512985, 0.9045560956001282, -0.7245833277702332], [-0.014005755074322224, 0.5506020784378052, -1.205862283706665, 0.8830478191375732, 0.2106158584356308, -0.9523411989212036, -0.24264249205589294, -1.4264289140701294, 0.5919209718704224, -0.035958368331193924, -1.3632915019989014, 1.9183968305587769, 0.8139184713363647, 0.6912352442741394, -1.1060786247253418, -1.3279885053634644, 0.8930712342262268, 0.45117470622062683, 0.3090292811393738, -1.5054609775543213, 0.38911503553390503, 0.678068220615387, 1.1125521659851074, -1.8905545473098755, -0.8393733501434326, 0.5803723931312561, -1.2545429468154907, -1.0913734436035156, 1.1080429553985596, -0.39175036549568176, -0.07283896952867508, -0.8099192976951599, 0.33949998021125793, -1.1176103353500366, -1.4063470363616943, 0.28770768642425537, 0.049175046384334564, 0.12445104867219925, 0.6032111048698425, -0.2768835127353668, 0.09747668355703354, 2.8831615447998047, 0.5362674593925476, -0.3953608572483063, -0.3291723132133484, 0.05519359931349754, 0.38147398829460144, -0.38365691900253296, 0.8168781995773315, 0.19143112003803253, -0.09978233277797699, 0.7715660333633423, -1.207231879234314, -0.9477736949920654, 0.5808735489845276, -1.8308717012405396, -0.7312027812004089, -0.6638030409812927, 0.005306030623614788, -1.8534966707229614, 2.191554546356201, -0.9073959589004517, 0.045727912336587906, 1.3797714710235596], [1.3902515172958374, 0.2877681851387024, 0.8251571655273438, -0.8646895885467529, -0.9645817279815674, -0.9521104097366333, -0.584136426448822, -0.06856338679790497, 1.1165497303009033, 0.9427222609519958, 0.485003262758255, 1.366707444190979, 0.194069504737854, -0.0018582851625978947, 0.1368139535188675, 0.7810993790626526, -0.23472657799720764, 0.1663742959499359, 0.09246938675642014, -0.828126847743988, -0.4625411927700043, 0.228130504488945, 0.023995762690901756, -0.8044687509536743, 0.8513607382774353, 1.8220280408859253, -0.02640298195183277, -0.6419013142585754, -0.6837194561958313, 0.4836875796318054, 1.5931085348129272, -0.1575937569141388, -0.10329674184322357, -0.39087799191474915, -0.01866118423640728, 0.5085265040397644, -0.36690378189086914, 0.3342169225215912, -1.0426881313323975, -1.6472065448760986, 0.45222100615501404, -0.29437538981437683, 0.14926353096961975, 0.6788806915283203, 0.36435237526893616, -2.1169023513793945, 2.054488182067871, -0.14674368500709534, 0.21495398879051208, -0.7817699909210205, -0.03650953993201256, -0.37467315793037415, 0.8780931234359741, -1.2165995836257935, -0.3227832615375519, 0.47851625084877014, 0.2769787907600403, -1.022487998008728, -0.10143566131591797, -1.5273079872131348, -0.5196540951728821, 0.49105024337768555, 0.48036089539527893, -0.6432353258132935], [1.0643186569213867, 0.1573655605316162, 0.3989911377429962, -0.3989602327346802, -0.3958275020122528, -0.0925508365035057, -0.13002899289131165, -0.7706928253173828, 0.17705638706684113, -1.1993063688278198, -1.79475736618042, 0.7261229753494263, -1.480637550354004, -0.8267199993133545, -1.700175166130066, 1.6424461603164673, -1.1598260402679443, -0.39095863699913025, 0.1107177585363388, -0.09072522073984146, -0.7566426396369934, 0.10090538114309311, -0.24051609635353088, -1.4592689275741577, 0.17699608206748962, -0.01208486221730709, 0.7569180130958557, 0.15776799619197845, -1.1716476678848267, 0.09810660779476166, -0.6337353587150574, 1.4744395017623901, 2.1917905807495117, -1.328479290008545, -0.6298107504844666, -1.6234604120254517, 0.8882924318313599, 0.4652755856513977, -1.5073851346969604, 0.09929847717285156, 0.13940604031085968, -1.9254206418991089, -0.46409040689468384, -1.2537846565246582, -1.074873447418213, -0.3547980487346649, -1.1564325094223022, -1.1073707342147827, 0.4640437960624695, 0.12283675372600555, -0.6612945795059204, 0.8419765830039978, -1.845766544342041, 0.9037477970123291, -0.6594877243041992, 0.039953432977199554, -0.43466809391975403, -0.5299076437950134, 1.7909115552902222, 1.0271244049072266, -2.0898196697235107, 0.41927847266197205, -0.018170377239584923, -0.1045914813876152], [-1.164932131767273, -0.7037194967269897, -1.587228775024414, 0.6902994513511658, -1.5271774530410767, 0.3830691874027252, 0.3414325416088104, -1.4421495199203491, 0.3521881699562073, 0.9412959218025208, -0.31095829606056213, -1.3687810897827148, -0.45291605591773987, 0.07037026435136795, -1.6491390466690063, -1.337557077407837, 0.648242712020874, -0.7836018800735474, -0.44464269280433655, -1.471659779548645, -0.9057486653327942, -0.9958221316337585, 0.1272449642419815, -0.40837687253952026, -1.0024700164794922, 0.4849955141544342, 0.3007809817790985, 2.7728934288024902, -0.09259457886219025, -0.48056861758232117, -1.4112541675567627, 1.3434946537017822, -1.4260202646255493, -0.33875882625579834, 0.5422768592834473, -0.46669861674308777, 0.733253002166748, 0.009383419528603554, 1.1042320728302002, 0.05153731256723404, 1.3069331645965576, 0.05970528721809387, 0.25717002153396606, 1.029707670211792, 1.2819838523864746, -0.6573824882507324, -1.5797295570373535, 0.4759277105331421, 1.3199628591537476, 0.4970085322856903, -0.03761213272809982, -1.409738540649414, 0.23624002933502197, -0.3585197627544403, 0.10868428647518158, 1.9872941970825195, -0.5601415634155273, 0.4162561595439911, -0.38593316078186035, 0.1400420367717743, -0.9092892408370972, -0.1304996907711029, 0.01321831438690424, 1.8224527835845947], [-0.644195556640625, -1.0056735277175903, -0.24654924869537354, -0.917263925075531, -0.6691651940345764, -0.7546720504760742, -0.01894083432853222, -0.7954978942871094, 0.08745197206735611, 0.41709983348846436, -0.12330224364995956, -1.3289775848388672, 1.5585880279541016, -0.14756856858730316, 0.37792280316352844, -0.0656026154756546, 0.961126983165741, -0.6831126809120178, -0.2244388312101364, -1.4061411619186401, -0.006935416720807552, 0.8358990550041199, -0.34292155504226685, 0.029259420931339264, 0.8082229495048523, 0.7499920725822449, 0.04334905743598938, -0.8742557168006897, -0.6649061441421509, 0.37855759263038635, 0.09986475855112076, 0.6769981384277344, 0.541644811630249, 0.31280314922332764, -1.080645203590393, -1.6872127056121826, 0.4831497073173523, -1.5054305791854858, 0.16438645124435425, -0.21051427721977234, 1.1081602573394775, -1.6361697912216187, -0.5646700263023376, 0.0018348683370277286, -0.6533657908439636, -0.6310514807701111, 0.24409230053424835, 0.08163636177778244, -0.7913898229598999, 0.6240264177322388, 0.5542711615562439, 0.1548151820898056, -0.9712766408920288, 0.8554818630218506, -1.0802942514419556, -0.1429230272769928, 0.6926798224449158, -0.40655508637428284, 0.5760378837585449, 1.463870882987976, 1.2525599002838135, -0.5070398449897766, -1.3451858758926392, 1.19225013256073], [0.15919016301631927, 0.18688024580478668, -1.0014837980270386, 1.726847529411316, 0.38677334785461426, 0.0354803241789341, 0.907650887966156, -0.6733133792877197, -0.4968298375606537, -1.0524959564208984, 0.9749950766563416, 1.222572922706604, 0.35948464274406433, -0.18336640298366547, -0.27849280834198, -0.15066400170326233, -1.0818127393722534, -0.0705007016658783, -0.1837126910686493, 0.021552419289946556, 0.7063227295875549, -1.5606335401535034, 0.9036967754364014, -2.3585498332977295, -0.07385601103305817, 0.13400501012802124, 0.2764730155467987, 0.6165434122085571, 0.5816645622253418, -0.1721956878900528, -0.1647203266620636, 0.7245743870735168, -0.9145128726959229, 1.7208539247512817, -0.25184014439582825, -0.3160243332386017, 1.144935131072998, 0.5599855780601501, -0.4319770038127899, -1.021064043045044, 0.0007565958076156676, -1.2204073667526245, 0.6891810894012451, -1.2428014278411865, 1.532334804534912, 0.40342918038368225, 0.3572349548339844, -0.7530184388160706, -0.2569214701652527, 2.183046579360962, -0.7019652128219604, -0.7051798701286316, -0.839887797832489, 0.18316343426704407, 1.1796687841415405, -0.061110299080610275, 1.035402536392212, 2.691915988922119, 0.08871069550514221, -1.2244155406951904, -0.03995512053370476, 0.4030817151069641, -0.24466168880462646, 0.594307541847229], [-0.2182890623807907, 0.6241633296012878, -1.34402596950531, -0.3454959988594055, 0.17152492702007294, 0.39375218749046326, 0.4111478626728058, 0.04674327000975609, 0.19074752926826477, -0.7779505848884583, 0.33213844895362854, 0.42779043316841125, 0.34812694787979126, -0.5515667796134949, -1.7312177419662476, 0.19756774604320526, 0.4045398235321045, 1.0847218036651611, -0.38402536511421204, -0.3499128222465515, -0.4235685467720032, 0.5410435795783997, -1.7539551258087158, -0.7823672890663147, -1.7989745140075684, 1.138404369354248, 0.40235936641693115, -1.2751163244247437, -1.4063719511032104, -0.6298544406890869, -1.902387261390686, -0.9246838688850403, 0.6569748520851135, 0.0600469633936882, 0.627687394618988, -0.23066525161266327, 0.10513556003570557, -1.4544121026992798, 1.5794473886489868, -0.6269994378089905, -1.1410058736801147, -1.729764699935913, -1.6328346729278564, 1.4146215915679932, 0.6434935331344604, 0.5232924818992615, 0.21307605504989624, -0.24444352090358734, 0.580935537815094, -0.3380068838596344, -0.1673399955034256, -0.532466471195221, 0.6211300492286682, -0.461355060338974, -1.1646647453308105, 1.512121319770813, 0.6430439949035645, 0.1150447204709053, -0.5267953872680664, 0.9950847625732422, 1.3892589807510376, -1.4811536073684692, 1.1646974086761475, 2.130697011947632], [-0.20089088380336761, -0.6437857747077942, -0.25129154324531555, 0.31864920258522034, 1.2365000247955322, -1.8039367198944092, -0.608400285243988, -0.06383766233921051, -0.26052385568618774, -0.5794777870178223, 0.5601658821105957, -0.5350218415260315, -0.48181647062301636, -1.1867820024490356, 1.4401073455810547, -1.549237608909607, 0.8572079539299011, -0.1970854550600052, 0.059172630310058594, 0.34533706307411194, -0.7700024843215942, -0.4723722040653229, 0.7240921854972839, 0.39015993475914, 0.4227462410926819, 0.6099779605865479, 0.6426460146903992, -1.6158210039138794, -0.27694350481033325, 1.1052944660186768, 1.2947593927383423, -0.4171395003795624, 0.2859490215778351, -1.2521193027496338, 2.133043050765991, 0.9134650826454163, -0.4187653362751007, 1.5920048952102661, -1.4593263864517212, 0.8325005769729614, 0.2820514738559723, -0.16633829474449158, -1.4721676111221313, 1.4285236597061157, 0.15074773132801056, 0.6213761568069458, -0.7767300605773926, -1.3958531618118286, -0.4232049286365509, -0.06922105699777603, 0.5067950487136841, -1.5383362770080566, -0.1726432889699936, -1.556966781616211, -1.348219633102417, -1.0763487815856934, -1.4000153541564941, -0.042040444910526276, -0.2573997974395752, 0.6227280497550964, 0.06156530976295471, 0.9188748598098755, -0.7669249176979065, 0.4750640094280243], [0.1079612746834755, -1.178126573562622, -1.6124368906021118, -0.4435425102710724, 1.640251636505127, -0.5651054978370667, -1.1549899578094482, 1.2996065616607666, 0.1634681224822998, -0.4159010052680969, -0.9754438996315002, -0.6858494877815247, 0.7455165386199951, 1.9224238395690918, -0.885148823261261, -2.2093441486358643, 1.3034950494766235, 0.4943821132183075, -0.9952561855316162, -0.8178654909133911, -0.0858796164393425, -1.4051443338394165, -0.1079452857375145, 1.3217014074325562, 0.7340992093086243, -0.5437924265861511, 0.24037297070026398, -1.4045060873031616, -1.2862335443496704, 1.2553026676177979, 1.0851073265075684, 0.7146494388580322, -0.5854995846748352, 0.4670238196849823, -0.5394169688224792, -0.37213248014450073, 1.668764591217041, 0.3959773778915405, 1.0851839780807495, -0.7585936784744263, 0.14268024265766144, 0.5737593173980713, 0.6770831942558289, -0.2158583253622055, 0.41819947957992554, 2.8917768001556396, -0.8110544085502625, -0.043142449110746384, 0.8445443511009216, 1.4315037727355957, -0.7222245335578918, 0.6511109471321106, -0.7435749769210815, 1.1839556694030762, 0.4684910476207733, -0.8095899224281311, 0.6809945702552795, 0.14198459684848785, 0.9730215668678284, 0.9940242171287537, 0.019625501707196236, -0.39610257744789124, 0.35362890362739563, -0.3251388967037201], [0.2647498846054077, 0.19207395613193512, 1.2627571821212769, 0.9068818092346191, 1.4093856811523438, 0.16930654644966125, 0.16087622940540314, -0.7886802554130554, 1.6609828472137451, 2.163288116455078, -0.530454158782959, -0.24946023523807526, -2.1794631481170654, -1.3394536972045898, 1.235533595085144, 1.2005093097686768, 2.157397985458374, -0.6892791986465454, 1.9194691181182861, -1.059552550315857, -0.29675954580307007, 2.465264320373535, 1.5198618173599243, 0.46419307589530945, -0.41636988520622253, 1.7253350019454956, -1.4790812730789185, 0.5098555088043213, -0.5258503556251526, -0.15122392773628235, 0.30319875478744507, -1.1645140647888184, -1.7922027111053467, -0.7596058249473572, -0.1999943107366562, 0.11325215548276901, 0.8944586515426636, -2.0548901557922363, -0.20794115960597992, -0.1013580858707428, -0.8111453652381897, 0.49718600511550903, -0.8104513883590698, 0.26826831698417664, 1.8679754734039307, -1.6815084218978882, -0.34717002511024475, 0.6759158968925476, -0.01341166254132986, 0.06730464100837708, 2.0164694786071777, -1.6480381488800049, -1.1637582778930664, -1.0714590549468994, -0.29300612211227417, 1.3094958066940308, -1.2657922506332397, -0.3643020987510681, -0.03359861299395561, 0.7888378500938416, 0.39069655537605286, 1.5351823568344116, -1.2428604364395142, -1.5882703065872192], [-0.03571806102991104, -1.073564887046814, -1.815059781074524, 0.9160414934158325, -0.32734420895576477, 0.11304952204227448, 0.6757737994194031, 0.23722201585769653, -0.24142947793006897, 0.9195752143859863, 1.115309238433838, 0.8240196704864502, -0.4775678217411041, -0.24594224989414215, -0.37144145369529724, 1.102490782737732, 1.6980494260787964, 1.700218677520752, -0.5501829385757446, 0.9958950281143188, 1.042339563369751, -2.3794546127319336, 0.21532344818115234, -0.3338446021080017, 0.7413772344589233, 0.5648232102394104, -0.683840274810791, -1.1319011449813843, 1.0639677047729492, 0.7928522229194641, -1.8548526763916016, -1.3518601655960083, 0.7621836066246033, 0.5114578008651733, -1.1132267713546753, -0.8797764182090759, 1.4902631044387817, 0.8339231610298157, -0.6892172694206238, -0.06872359663248062, -0.11243068426847458, 1.7412610054016113, 0.19486260414123535, 0.2793372869491577, -0.15811365842819214, 0.6353474259376526, 0.03743002936244011, -1.147719383239746, -1.7263360023498535, -2.2282769680023193, 1.5247366428375244, 0.05290522053837776, -1.8114608526229858, 2.764969825744629, -2.0057313442230225, -2.5473906993865967, 0.31401166319847107, 1.0587576627731323, -0.2155531495809555, -1.966373324394226, 1.0882315635681152, -0.3330991268157959, 0.4295791685581207, 1.0532559156417847], [1.3314759731292725, -0.09294534474611282, -1.146122694015503, 0.7311992645263672, -1.3216636180877686, -1.1917195320129395, 1.527401328086853, -2.354694366455078, -0.4006604552268982, 1.4712224006652832, -0.4284450113773346, -0.9941515922546387, -0.06022915244102478, 1.0017900466918945, 1.276907205581665, 0.4696454107761383, -1.7351016998291016, 0.38780540227890015, 0.15682978928089142, 1.4356855154037476, -2.937831163406372, -0.3299386501312256, 0.8619300127029419, 0.7864987850189209, -2.2766144275665283, 0.276741623878479, 0.2597939968109131, -1.7026475667953491, 0.6308276057243347, -0.8152629733085632, -0.111618772149086, 1.2469916343688965, -0.0194556787610054, 1.6148566007614136, -0.7180173397064209, 0.4233763515949249, -0.011768144555389881, -0.7212904691696167, -0.8066465854644775, 0.576575517654419, 1.4207721948623657, 0.5613623857498169, 0.3526446521282196, 1.6739797592163086, -0.7857711315155029, -1.4698143005371094, 0.5505262017250061, 0.038728538900613785, -2.698176622390747, -1.776882529258728, 1.0223416090011597, 2.1203932762145996, 0.7668820023536682, 1.204249620437622, -0.7466263771057129, 0.09431206434965134, -0.3896442949771881, -1.042178988456726, -0.45071473717689514, 0.5224319696426392, 1.1205532550811768, -0.1801530420780182, 0.8683047294616699, -0.09445953369140625], [-1.6519931554794312, 0.43334391713142395, 0.3418201208114624, -1.2352294921875, 1.606021523475647, 2.830514430999756, 0.7551843523979187, -1.5831189155578613, -1.846557855606079, -0.6126821637153625, 0.7625900506973267, 0.3338198959827423, -1.4220203161239624, -0.4317198395729065, 0.7712830305099487, -1.3278396129608154, 0.06977882236242294, -0.8437961935997009, -0.2237006574869156, 1.2278175354003906, -0.3503439128398895, 1.16797935962677, -2.032290458679199, 0.8574094772338867, -0.6093283295631409, 0.549589991569519, -0.07673503458499908, 1.3739466667175293, -1.120009183883667, -2.128448247909546, -0.7365496158599854, 0.1930733621120453, -0.14913791418075562, -0.638110876083374, -1.1031662225723267, -0.9460402727127075, -0.10191325843334198, -1.1173428297042847, 1.645995855331421, -0.8436457514762878, 0.6300775408744812, 0.14969636499881744, 1.3287769556045532, 0.48808351159095764, 0.11859120428562164, 0.9408154487609863, -1.5467908382415771, -1.2999571561813354, 1.0289584398269653, 0.6877345442771912, -1.2817180156707764, -2.348128080368042, 0.4521540403366089, 1.2082269191741943, 1.0620301961898804, 0.674808919429779, -1.0021013021469116, 0.31960710883140564, -0.26228243112564087, 0.16107214987277985, -0.10824555903673172, 0.45211589336395264, -0.4424942433834076, -1.3263286352157593], [0.9523817300796509, 2.43459153175354, 0.4137382209300995, -0.3668740689754486, -0.27183064818382263, 1.5355045795440674, -0.6978620886802673, 0.17790278792381287, 1.1175096035003662, 1.1658399105072021, -0.4236833453178406, -1.258467197418213, 0.6484071612358093, -1.5410795211791992, 0.10892496258020401, 0.2837430238723755, 0.2083519548177719, -1.1359635591506958, 0.9007324576377869, 0.7677565217018127, -1.5918456315994263, 1.002771258354187, -1.0453622341156006, 0.12294268608093262, 0.7248280644416809, 0.05123598873615265, 0.02300250716507435, 0.06794432550668716, 0.35087889432907104, 0.15493658185005188, 1.0689654350280762, -1.486581802368164, 0.09467566013336182, 1.4613969326019287, -2.489609479904175, -0.06927413493394852, -0.4038732647895813, -1.295669674873352, 0.1030103787779808, -0.724543035030365, -2.035855531692505, 0.08315884321928024, 1.403908133506775, 0.32589486241340637, -0.029762543737888336, -0.9672757983207703, 0.06365068256855011, 0.07312550395727158, 0.40272822976112366, -1.1458756923675537, 0.31452661752700806, 0.49173447489738464, 0.2193344682455063, -0.99244225025177, 0.7912675738334656, 0.7695441842079163, 0.9438881874084473, 1.2353854179382324, 0.4238187372684479, -0.4292285740375519, 0.29821500182151794, -0.6337391138076782, 0.204397514462471, -0.33107054233551025]], [[-0.9949422478675842, 0.2459852397441864, 1.4011441469192505, -0.4544806182384491, 1.6836357116699219, 0.43121233582496643, 1.7454942464828491, 1.5476747751235962, -0.09927249699831009, 0.4284362196922302, -0.9569141864776611, -0.4550188183784485, 0.34702858328819275, 0.4070546627044678, -0.20722752809524536, -0.8545247912406921, 2.2298429012298584, 1.334288477897644, 0.5154473185539246, -0.25196269154548645, -0.7519041895866394, 1.0532323122024536, 0.5065150856971741, 1.0464574098587036, 0.18798570334911346, 0.6984654068946838, 1.7496424913406372, -0.3099786639213562, -1.1198660135269165, 0.9405900239944458, 0.5338239073753357, 1.9220249652862549, 1.173625111579895, 1.790632724761963, 0.15807189047336578, 0.8112704157829285, -0.7717803120613098, -2.2176334857940674, -0.15446162223815918, -0.22200097143650055, 0.18319638073444366, -0.5729701519012451, 0.5423130989074707, 0.27359718084335327, 0.46528688073158264, 2.242260694503784, -0.8552306890487671, 4.084046363830566, 0.8815862536430359, 0.10341586917638779, 0.10506941378116608, -0.2846904993057251, 0.485767662525177, -1.649118423461914, 0.7382707595825195, -1.7266148328781128, -0.14403890073299408, 1.3906160593032837, 1.36977219581604, 1.2746858596801758, -0.4066186845302582, -1.3265650272369385, 0.5411103367805481, -0.6360713243484497], [0.21098098158836365, 0.24085049331188202, 0.23367248475551605, -0.9740201234817505, -1.7144356966018677, -0.812390148639679, 1.394023060798645, -1.2093194723129272, 0.15278778970241547, 0.4472920894622803, 1.5988929271697998, 0.8746757507324219, 1.3572951555252075, 1.109328269958496, -0.661304235458374, 0.5957000851631165, -1.245664358139038, 2.258204936981201, 1.0852891206741333, -1.4209998846054077, -0.9300857186317444, 0.653425395488739, -0.9180313348770142, 0.23817779123783112, 0.6896769404411316, -0.1471548229455948, -2.2510392665863037, -1.740260124206543, 0.691807746887207, -0.6976827383041382, -0.007262126076966524, 0.3225676119327545, -1.2754980325698853, 0.34659844636917114, -1.9381242990493774, -0.2959845960140228, 0.5817258954048157, 1.9521313905715942, 1.3140645027160645, 1.974884033203125, 1.299557089805603, 0.8265095949172974, 0.2727760076522827, 1.7288587093353271, -1.0197010040283203, -0.8883903622627258, -0.9943159818649292, -1.197710633277893, -1.0566301345825195, 0.21752868592739105, -1.4967166185379028, -0.2534613013267517, -0.2641121447086334, 0.1319870501756668, -0.5063642859458923, -0.7350525259971619, -0.2908066511154175, 0.7410039901733398, 0.23109814524650574, 0.05830013379454613, -0.3566206991672516, 1.1751350164413452, -1.4448579549789429, -2.2373206615448], [-1.81161367893219, 1.1606078147888184, 0.0589078851044178, -1.2420765161514282, 0.6462399363517761, 0.4838264584541321, 0.4488161504268646, 0.018359245732426643, -0.26613649725914, 1.3747507333755493, 0.29387837648391724, 0.011442327871918678, -0.7330398559570312, 0.13389788568019867, 0.5513520836830139, -1.4916012287139893, 0.4338633120059967, -0.38700631260871887, -0.9239317178726196, 0.5368218421936035, 2.247563362121582, -1.1154927015304565, -0.6039007306098938, -1.8673217296600342, 0.9551874995231628, 0.661564290523529, -0.8017662167549133, 1.4892102479934692, 0.8410003185272217, -0.07781556993722916, 0.441607803106308, 0.2877582609653473, 1.3217461109161377, -1.9957720041275024, 1.3739194869995117, -1.4815560579299927, 1.519698977470398, 0.555057168006897, 0.14802183210849762, -0.8140326142311096, -0.22385726869106293, 0.4982047975063324, 0.7954748272895813, 1.5949511528015137, -0.5223003625869751, -0.5655446648597717, -0.35807284712791443, -0.3417026400566101, 0.20218636095523834, -0.6382884979248047, 0.4165228307247162, -1.3043243885040283, -0.547444224357605, 0.5052507519721985, -1.4276431798934937, -0.791780948638916, 0.44584691524505615, -0.04314348101615906, -0.3885185122489929, 1.6931626796722412, -0.17536792159080505, 1.0761502981185913, 0.15714715421199799, -0.8176491260528564], [-0.41951075196266174, 0.38985002040863037, 0.5920450091362, -1.0725243091583252, -0.3523346483707428, 1.2688140869140625, 0.28882208466529846, -1.4626110792160034, -0.11964297294616699, 0.17225807905197144, -0.9195331931114197, -1.2197753190994263, 0.3635430634021759, -0.21246905624866486, 1.8911337852478027, -1.0999797582626343, 0.1906764954328537, 0.4076322019100189, -1.2097251415252686, 0.50749671459198, -0.7401384711265564, 0.8895146250724792, -1.5856051445007324, -0.3732719123363495, 0.12643831968307495, 0.714486837387085, -1.043555498123169, -1.603877067565918, 0.6515832543373108, -0.44418251514434814, 0.725369393825531, -1.3824952840805054, -0.9134519100189209, -0.994240403175354, -0.87492835521698, 0.4787808358669281, -0.8759537935256958, 0.6093623638153076, -0.8604093790054321, -1.2868938446044922, 0.3006233274936676, 0.37618133425712585, 2.0287671089172363, -2.600975751876831, -0.8406355977058411, 0.3101212680339813, -0.28808820247650146, 1.7563748359680176, -0.4817017912864685, 0.2488914579153061, -0.536908745765686, -2.164926528930664, -2.7600533962249756, -1.6843734979629517, -2.469153881072998, 0.5822861790657043, -0.11048127710819244, -1.9656671285629272, 0.23983389139175415, 1.2580015659332275, -0.7392769455909729, -1.2642344236373901, -0.8005959391593933, -0.0036191255785524845], [1.1959726810455322, 0.565572202205658, 0.44971656799316406, 0.09543583542108536, -0.6542319059371948, -1.756625771522522, 0.35055941343307495, 0.3217517137527466, -0.754188597202301, -0.19716797769069672, -0.9955018162727356, -1.0251888036727905, 0.6317073702812195, -0.7034367322921753, 0.011991696432232857, -0.4997899830341339, -0.3730678856372833, 1.1648684740066528, -2.0422894954681396, -0.5738495588302612, 2.188246726989746, 0.3143080472946167, 0.2191867083311081, 0.7311874032020569, 1.145680546760559, 0.4063992202281952, -0.6386862397193909, -0.24890801310539246, 1.692233920097351, -0.6711574196815491, -1.6659809350967407, 0.13228286802768707, 0.41366660594940186, 0.45507943630218506, 0.15737532079219818, -0.8172154426574707, 1.289679765701294, 1.9054129123687744, -0.9669414758682251, 1.736833095550537, 0.5594533681869507, -1.1902936697006226, 0.04632464051246643, 0.18741483986377716, 0.7083494663238525, 0.485451340675354, 0.8939657211303711, 0.33009618520736694, 1.2471810579299927, -0.17968526482582092, 0.7403998374938965, -1.5131500959396362, -0.07777278125286102, 2.002448320388794, -1.0774863958358765, 2.9384682178497314, 2.466109275817871, -0.04586417227983475, -1.6524152755737305, 0.3369765281677246, -1.5479536056518555, 1.0719578266143799, -1.0868351459503174, -0.21420563757419586], [-1.8068833351135254, -0.30993151664733887, -0.06765174865722656, 0.4957166016101837, 1.0847066640853882, 1.6154234409332275, 0.15538626909255981, -0.8665551543235779, 0.9136213064193726, -1.3917632102966309, 1.0826153755187988, -0.930596113204956, -0.25413280725479126, -0.952540397644043, -1.4854835271835327, -0.6652039289474487, 0.22094596922397614, -1.3744066953659058, 1.199278473854065, -0.49366679787635803, -0.1952829509973526, -0.439778208732605, 0.011491550132632256, 0.3002108633518219, 0.4214594066143036, -0.24837364256381989, 2.284590482711792, 1.0506917238235474, 0.6144401431083679, -0.14106470346450806, -0.3301955759525299, 1.4958617687225342, -1.3265373706817627, -0.5747475624084473, -0.48009398579597473, 0.9503005743026733, 0.7457674741744995, 1.2579175233840942, -0.5754433274269104, 0.5606024265289307, 0.29328131675720215, 0.1717132180929184, 0.6180459856987, 1.126302719116211, 0.5262008905410767, 0.7493679523468018, -1.0848180055618286, 0.2372211366891861, -0.7182983756065369, -0.12202909588813782, 0.7903457880020142, -2.7705063819885254, -0.7041110992431641, 1.0670990943908691, 0.9895842671394348, -1.3876969814300537, -0.738437294960022, -1.6065428256988525, 1.1997151374816895, -0.28850501775741577, -0.12408001720905304, -1.4494863748550415, -0.2846980094909668, -0.5099372267723083], [-2.7688167095184326, -1.5255074501037598, -2.981712818145752, -0.345665842294693, -0.4203084409236908, 1.4699851274490356, -2.4547488689422607, 1.3023018836975098, 0.3893396258354187, 0.717572033405304, -0.18033918738365173, -0.6192570328712463, 1.4030303955078125, -1.691902756690979, 1.2195693254470825, 1.0319222211837769, 0.5066291093826294, -0.23757584393024445, 0.764606237411499, 0.38164883852005005, -1.4347765445709229, 0.28124332427978516, 0.03216835856437683, 0.47405678033828735, 0.9497878551483154, 1.0308243036270142, 0.4952124059200287, -0.16008701920509338, -0.06109614297747612, -0.5112041234970093, -1.2474229335784912, 0.633922278881073, 0.3996618986129761, 0.42950373888015747, 0.564572811126709, 0.1912776231765747, -0.7335266470909119, -0.3972913920879364, 0.14653639495372772, 1.6647013425827026, 0.6974106431007385, 0.4111306369304657, 0.08065754175186157, 0.11127801239490509, -0.03214649856090546, -0.957160234451294, 2.353066921234131, -1.2476001977920532, 0.19646140933036804, -0.7034879922866821, -0.8804206252098083, 2.0259010791778564, 0.3334009051322937, -1.869641661643982, 1.0814388990402222, 2.511021614074707, 0.7050632834434509, -1.6869211196899414, 0.8198149800300598, -0.3135402798652649, 0.7766656875610352, 0.9759955406188965, 0.031153056770563126, -1.301391363143921], [0.002882617060095072, -0.864704430103302, 1.0666483640670776, 0.14152075350284576, -0.3517150282859802, -1.7494832277297974, -1.7143827676773071, 0.16470597684383392, 0.6715195178985596, 0.4938163459300995, 1.984713077545166, 0.7521482706069946, -1.3559921979904175, 2.002753734588623, 1.3838257789611816, -0.5151078104972839, -0.42506593465805054, 1.9816545248031616, 1.0822254419326782, 1.1243948936462402, -1.0778697729110718, -0.3094760477542877, -0.993646502494812, -0.6530218124389648, -0.5713019371032715, 0.6459251046180725, 0.7571723461151123, -0.5417315363883972, 1.9603278636932373, 0.05084004998207092, -0.19311270117759705, 0.648151695728302, -2.0143375396728516, 0.18532878160476685, 0.38367971777915955, -0.6179466247558594, 0.8181671500205994, 0.7857585549354553, 0.9846933484077454, 0.971111536026001, 1.632079005241394, 0.5985841751098633, -0.5407556891441345, 0.05072811245918274, 0.428242564201355, -0.3267712891101837, 1.4248261451721191, 1.6348756551742554, -0.9736685752868652, -0.3765887916088104, 0.10363530367612839, -0.0993359312415123, -1.3292641639709473, 1.08529531955719, 1.2003169059753418, 1.078646183013916, -1.8283263444900513, 0.10772042721509933, -1.8693633079528809, 0.07242367416620255, 1.5743521451950073, 0.06698082387447357, -0.9772310256958008, -1.1192692518234253], [0.7059117555618286, 1.3650728464126587, 0.28618717193603516, 0.5088871121406555, 0.8336738348007202, 0.23915590345859528, -0.26974424719810486, 0.8184224963188171, 1.505191683769226, -0.40397271513938904, 1.1331872940063477, -1.270864725112915, -1.5615066289901733, 0.12111131846904755, 1.1249805688858032, -0.21252119541168213, 0.34929338097572327, 0.7565247416496277, 0.20155197381973267, 0.6535778641700745, 0.964544415473938, 0.7209293842315674, -1.6137663125991821, -0.3911270201206207, -0.04526209831237793, 1.78336501121521, 0.40928569436073303, 0.8372264504432678, 0.8326970338821411, -0.24878205358982086, 0.861754298210144, 0.9152221083641052, 1.0039196014404297, -1.9283359050750732, 1.1045249700546265, 0.11166827380657196, -0.7552402019500732, 0.05986257269978523, 0.5467034578323364, 0.33502137660980225, 0.41275614500045776, -1.6374751329421997, -0.6176034212112427, 1.6006019115447998, 0.7669300436973572, 0.3531596064567566, -1.3154540061950684, 0.09189524501562119, 1.0507184267044067, 0.378611296415329, -0.4717326760292053, 0.2883109152317047, 0.5816047787666321, -0.8719629645347595, -0.9425174593925476, 1.3953089714050293, -0.5076507329940796, 0.42582932114601135, -1.7403837442398071, 1.2024545669555664, -0.6974551677703857, -0.38714468479156494, -0.007289333269000053, -0.6776732802391052], [0.8761478066444397, 0.9179368615150452, -0.3892427086830139, 0.38819608092308044, -0.24317553639411926, 1.5437973737716675, -0.0409938208758831, 1.3991304636001587, 0.26970353722572327, 1.1454713344573975, 0.9003570675849915, 0.9119595289230347, -0.09370800107717514, -0.23791080713272095, -1.8611705303192139, 0.3939945101737976, 0.5758221745491028, -1.3005554676055908, -1.9832334518432617, 0.01373556163161993, -0.7591463923454285, 0.5097705125808716, 3.1918563842773438, -0.03204956278204918, 0.8679468035697937, 0.8759178519248962, -0.888192355632782, -1.0698561668395996, -0.37862536311149597, 1.4798457622528076, 0.16385719180107117, 2.511021852493286, 0.7484445571899414, 0.2630501091480255, -0.8672106266021729, 1.0716248750686646, -2.2984535694122314, -0.03921768441796303, 0.08124884217977524, 2.1735799312591553, 1.5197604894638062, 0.13122470676898956, -0.4808948040008545, 1.1237748861312866, 0.12382945418357849, -0.2645317018032074, 2.6415815353393555, -0.9568057060241699, 0.8518780469894409, -0.052272334694862366, 1.7058801651000977, -1.5874673128128052, -1.5836211442947388, -0.6561729311943054, -0.7888287901878357, 0.071555495262146, -1.1951708793640137, 0.25761300325393677, -0.6694763898849487, 0.020138133317232132, 1.9906878471374512, -0.8811867237091064, 0.032434165477752686, -0.5063203573226929], [0.6838266849517822, 0.4255801737308502, -0.0633796900510788, -1.1423860788345337, 0.3318638503551483, 0.28248080611228943, 0.14936061203479767, 0.5419828295707703, 2.39859938621521, -1.5377508401870728, -1.0753540992736816, -2.3561952114105225, -0.4894341230392456, -0.40394553542137146, -2.0389368534088135, 0.5671156048774719, -0.6018934845924377, 1.7712271213531494, -0.39596670866012573, 0.2881534695625305, -0.2742651402950287, -1.4285588264465332, 1.4593515396118164, 0.7640348672866821, -1.3513542413711548, -0.53242027759552, 1.7638840675354004, -1.6109514236450195, -1.3000797033309937, -0.9603842496871948, -0.9679659008979797, -0.9549384117126465, 0.47863778471946716, -0.4895903468132019, -1.1977870464324951, -0.6100513339042664, 1.9848742485046387, 1.8488117456436157, -0.024742895737290382, -0.397635281085968, 0.15958072245121002, -1.809446096420288, -0.3610536754131317, -1.3125550746917725, 0.5120975375175476, -1.0206011533737183, -0.24239253997802734, -0.23221011459827423, -0.07764500379562378, 0.32677483558654785, 1.9331508874893188, -0.8018123507499695, -0.1304389238357544, 0.7450947761535645, 0.29860660433769226, 0.8844764232635498, 0.9891070127487183, -1.2468699216842651, 0.5373407602310181, -0.3454706370830536, 0.42865875363349915, 1.3253849744796753, 0.47495388984680176, 0.9957704544067383], [-1.663960576057434, 0.4914518892765045, -0.1215878576040268, 0.1354624480009079, -0.29386216402053833, 0.5332263112068176, -0.22131098806858063, -1.6884405612945557, -0.23604445159435272, -0.5627837777137756, -0.520246148109436, 0.297910213470459, 1.3924959897994995, 0.14897790551185608, 3.2605807781219482, 1.429301381111145, -0.38803690671920776, -0.7367172241210938, 1.7333564758300781, 1.3557054996490479, 1.4565794467926025, 1.6431108713150024, 1.2034859657287598, -0.5533969402313232, -0.5922225117683411, -0.5556549429893494, -0.5952415466308594, -0.8442329168319702, -1.4185744524002075, -2.1255762577056885, -0.24991244077682495, 0.3844072222709656, 1.7590711116790771, 1.2489346265792847, -1.1197651624679565, -0.21775038540363312, 0.36170560121536255, -0.5576170086860657, -1.811985969543457, 0.29704222083091736, 0.06835416704416275, 0.18753619492053986, 0.2992722690105438, -0.9011162519454956, 0.48546966910362244, 0.4114774465560913, -0.25288155674934387, -0.9028388261795044, -1.0903247594833374, 0.48575863242149353, 0.2023448944091797, -1.3559317588806152, 0.08901496231555939, 0.41917628049850464, -2.0914831161499023, 0.20074203610420227, 0.04114663600921631, -2.1080162525177, 0.27016398310661316, -0.14700336754322052, -0.25159579515457153, -1.2014049291610718, -0.4761340022087097, 0.39365988969802856], [0.16246488690376282, -1.53462815284729, 0.19442413747310638, -0.36049413681030273, 1.2785930633544922, 1.882266879081726, -1.898266077041626, 1.1291882991790771, 0.7183877229690552, 0.9184252619743347, 1.3555876016616821, -0.9627693295478821, 2.3679888248443604, 2.805894374847412, -1.1261742115020752, 1.8871532678604126, 0.6928099989891052, 1.7143551111221313, 1.1959474086761475, 0.2656201422214508, 0.36159852147102356, -0.21622073650360107, -0.1804133504629135, -0.22679713368415833, -0.1609942764043808, 1.558529257774353, 1.1254196166992188, -1.6394327878952026, -0.7811004519462585, 2.28682804107666, 0.463192880153656, -0.0014380449429154396, 0.5390143394470215, -0.25313180685043335, 1.1938717365264893, 0.2632448375225067, 0.23393748700618744, 0.28357020020484924, 1.1480839252471924, 0.5683402419090271, -1.520421028137207, 0.15801052749156952, -1.9505385160446167, 0.9603298306465149, -0.17339468002319336, -1.9048455953598022, -0.5345818996429443, 0.6286154389381409, -0.6464714407920837, -0.2348410040140152, 0.8377131223678589, 2.19988751411438, 0.3175088167190552, 1.3408294916152954, -0.5028790831565857, 0.05547294765710831, -0.35822173953056335, 0.14928652346134186, 2.4421069622039795, -0.4144344925880432, 0.7964881658554077, 0.6770954728126526, -0.07665497064590454, -1.306041955947876], [-0.999545156955719, -0.21853794157505035, 1.7537846565246582, 1.0589568614959717, -0.44105085730552673, -0.0831449031829834, -1.6981562376022339, -0.8393661975860596, 0.4890024662017822, -0.7549655437469482, -0.3033987283706665, 0.4937599301338196, 0.7913530468940735, 2.170301914215088, 0.15170833468437195, -0.6426019072532654, 0.4161474108695984, 1.6073423624038696, 0.4746098518371582, 0.5923330187797546, -0.5841712355613708, -0.07413838803768158, 0.712123453617096, -0.8927935361862183, 0.34227070212364197, 1.7610273361206055, 0.4661853611469269, -0.16913814842700958, -0.016878660768270493, 1.4530144929885864, -0.3085763454437256, -1.5659085512161255, -1.5489190816879272, 1.1387463808059692, 0.951460063457489, 0.7105292677879333, 0.9596329927444458, 0.397687703371048, 0.6479960083961487, -0.5224999785423279, -0.9608180522918701, 1.4919177293777466, -1.22390878200531, 1.8308392763137817, 1.111789345741272, 0.9957287311553955, 1.2631624937057495, -1.119734287261963, -0.5587321519851685, -0.2767731547355652, 1.4767022132873535, -0.3281194269657135, 0.33851611614227295, -0.05591706559062004, -0.10544466972351074, -1.3003122806549072, 0.40424713492393494, 0.34716033935546875, 0.0005554936942644417, 0.027857786044478416, 0.7195689678192139, -0.12106563150882721, 0.7784875631332397, -0.26750680804252625], [0.34078869223594666, -1.1278860569000244, -0.34431684017181396, 0.33375272154808044, -0.23218201100826263, -0.38088351488113403, -0.19302481412887573, -0.67910236120224, 0.333161860704422, 1.0801100730895996, 1.3827382326126099, -0.12333495169878006, -0.7114970088005066, -0.5028367042541504, -0.016308678314089775, 0.28417858481407166, -0.06915165483951569, 0.7062626481056213, -0.4138045012950897, 0.26622676849365234, -0.07592765986919403, -1.0281035900115967, -1.0795081853866577, 1.396469235420227, 1.328722357749939, 1.4061124324798584, 2.072958469390869, 1.1952983140945435, 1.5474448204040527, -0.6548864841461182, 0.10407040268182755, -0.5774788856506348, 0.5669749975204468, -2.0089166164398193, -0.5382843017578125, 0.14618465304374695, -1.8783812522888184, 1.3326727151870728, -1.0008397102355957, 0.3384149968624115, -1.254077672958374, -0.23190315067768097, 0.8438881039619446, 0.7858874201774597, -1.7356771230697632, -0.14569906890392303, -0.0969250500202179, -0.31512632966041565, 0.2610941231250763, 1.4403445720672607, 0.04331548139452934, 1.3358699083328247, -1.1144789457321167, 1.2040420770645142, -0.21883758902549744, -0.7105646729469299, 0.3006296157836914, -0.2047630399465561, 0.21019475162029266, 0.5008648037910461, -0.37860581278800964, -0.827191174030304, 0.6545153260231018, 1.8619004487991333], [0.6844220161437988, -0.601478099822998, -1.9354526996612549, 0.7072392702102661, 2.4177730083465576, -0.1741953343153, 0.017825864255428314, -1.924646019935608, -0.4473404884338379, -0.3469984829425812, -0.22206801176071167, 1.3491653203964233, -0.7252739071846008, -1.490857720375061, 0.5173566937446594, -0.46234366297721863, -0.8244308829307556, -0.351791650056839, -0.5212371945381165, -0.2439795583486557, 0.6454957127571106, 0.20403780043125153, -1.3567415475845337, 0.1523982733488083, 0.568221926689148, -0.9791455268859863, 2.0667684078216553, 1.6556988954544067, -0.05611753091216087, -1.7267931699752808, -1.6222221851348877, -0.044826630502939224, 0.4708004891872406, 0.8068974614143372, -0.6192606091499329, 0.4646504521369934, 0.17278769612312317, 0.8069787621498108, 0.3223337233066559, -1.2414441108703613, 1.0134668350219727, 0.3088182806968689, 0.9873359799385071, -0.7084733843803406, 0.7750965356826782, -0.7780454754829407, -0.10328322649002075, -0.6969770789146423, -0.36092156171798706, 0.2765319049358368, -1.226103663444519, -1.1618893146514893, 1.3858983516693115, -0.8963788747787476, 0.5734605193138123, -1.8046848773956299, 0.4474916458129883, 0.8282909393310547, -0.5270226001739502, -0.4593992531299591, -0.5960585474967957, 0.7562808990478516, 1.54812490940094, 0.6049963235855103]]], [[[-0.0035043407697230577, -1.965027093887329, 1.2606388330459595, 1.3275344371795654, 1.650688648223877, 0.1143346056342125, -0.9443110227584839, 0.5807506442070007, 0.5995644927024841, -0.2808294892311096, -0.8244168162345886, 0.6921457648277283, 0.6840089559555054, -0.8769158124923706, 0.2884831726551056, -0.3373187184333801, -0.9157685041427612, -0.31728702783584595, -0.06904439628124237, 1.4468048810958862, -1.6144863367080688, 0.34549862146377563, -0.025299230590462685, 0.648374080657959, -1.056593418121338, -1.5809987783432007, 2.0338704586029053, -1.170409083366394, -0.9321979880332947, -1.7077937126159668, 0.9405695796012878, 0.1184430792927742, -1.8205853700637817, 0.08880840241909027, 2.637526273727417, -0.6606269478797913, 0.3151310682296753, -0.05196134373545647, 0.11958079040050507, -2.024534225463867, -0.9178933501243591, -1.8885245323181152, 0.09228202700614929, 1.8156194686889648, -1.1930264234542847, 0.507929265499115, 0.9763310551643372, 0.6355945467948914, -0.10421761870384216, -0.11542994529008865, -0.6109566688537598, 2.681284189224243, 0.2178967297077179, 0.29970431327819824, 0.14330139756202698, 1.074090838432312, 0.5359312891960144, 0.507987916469574, 2.487351417541504, -0.10783462226390839, 0.9274989366531372, 1.0424141883850098, -0.9954321980476379, -0.6379263997077942], [1.3108704090118408, -1.2243235111236572, 1.6122045516967773, -0.24193896353244781, -0.36194270849227905, -0.5510807633399963, 0.6233330965042114, -1.3176060914993286, -0.3624251186847687, 0.004635340068489313, 1.1928555965423584, 0.7469760775566101, 1.0801165103912354, -0.22155746817588806, 0.013117793016135693, -0.13768082857131958, 1.032599925994873, 0.5614415407180786, 0.8287119269371033, 0.601574718952179, 0.01698227971792221, 0.21819399297237396, 1.5766119956970215, 0.39284223318099976, 1.6074762344360352, -0.8411681056022644, -0.20298825204372406, -0.29710397124290466, -2.039069890975952, 0.6379614472389221, -1.0197511911392212, 0.23160827159881592, 0.5271843671798706, 0.3726716935634613, 0.22476615011692047, -0.09135009348392487, 0.7333868741989136, -0.4734139144420624, 1.1535216569900513, 0.15058328211307526, -0.8088114261627197, -1.2447925806045532, 1.5490068197250366, 0.2821041941642761, -1.1111292839050293, 0.020790893584489822, 0.518920361995697, 1.4143751859664917, -2.093676805496216, -2.9201807975769043, 0.8187131285667419, -0.5364447236061096, -1.880205512046814, -0.746749222278595, -2.0526070594787598, -0.6235450506210327, -0.527405858039856, -0.07565044611692429, 0.7587063312530518, -2.0147511959075928, -2.4645676612854004, -0.36327555775642395, -0.6343874335289001, 1.0505356788635254], [-0.4085988402366638, 1.5051857233047485, -0.27551326155662537, -0.11363039165735245, 0.4451833665370941, 0.7445854544639587, 1.666689157485962, 0.020694434642791748, -0.08455429971218109, -0.36859259009361267, 1.351860523223877, -1.3851603269577026, 1.0551657676696777, -1.014696478843689, -0.5381258130073547, 0.9790154099464417, -0.16129888594150543, -1.1924182176589966, -0.1367144137620926, -0.4511609375476837, 0.004396640229970217, 0.18571221828460693, 0.9592229723930359, 0.3871864974498749, -1.0939035415649414, -1.6969704627990723, -0.5550876259803772, 1.3889755010604858, 0.3473759889602661, 1.8121811151504517, -0.9187291264533997, -0.3163723945617676, -1.526667594909668, 0.37734365463256836, 0.678606390953064, 1.1964713335037231, -1.3893802165985107, -1.459288477897644, 0.5435578227043152, -0.3119731843471527, -1.8747472763061523, -0.42974740266799927, -0.06516040861606598, -1.6971529722213745, -0.6920297145843506, 0.08180849999189377, -0.5225794911384583, 0.199692502617836, 0.14400488138198853, -0.8540441393852234, 0.2855721414089203, -1.1976066827774048, -1.134508728981018, 1.2406741380691528, -0.7470465898513794, 1.319879174232483, 0.2070544958114624, 1.477077603340149, 0.13988742232322693, 0.812926709651947, -0.42501407861709595, 1.647939682006836, 1.7446939945220947, -0.6056336760520935], [0.7352038025856018, -0.07419486343860626, 0.9678552746772766, -1.4382823705673218, 0.11575613915920258, 0.914422869682312, 0.5232232213020325, -0.6192014217376709, -2.325732469558716, -0.27987203001976013, -0.38225990533828735, -0.6179214715957642, -0.41666659712791443, 1.4822710752487183, 0.6247348189353943, -0.8270338177680969, 1.4600428342819214, -1.1892907619476318, -0.2851915955543518, 1.3389583826065063, 1.1434415578842163, -0.32481494545936584, 1.7702327966690063, 0.7706169486045837, -0.7946378588676453, 2.0185766220092773, -0.602687418460846, 0.9978245496749878, -1.0006699562072754, -2.2639753818511963, 0.5127293467521667, -1.1796588897705078, 2.2652649879455566, -0.1373605728149414, 1.3473178148269653, 0.8474147319793701, 1.7817434072494507, 0.1817164570093155, 2.3410401344299316, -0.22765956819057465, -0.3439977765083313, -0.5381550192832947, -0.5182314515113831, 1.2690354585647583, -1.5594855546951294, -2.7472481727600098, -1.752937912940979, -0.8820528984069824, -2.0867507457733154, -0.7046137452125549, -1.2275532484054565, 0.5269913673400879, 0.33500802516937256, -0.2139955759048462, 1.0709097385406494, 0.2052765041589737, -1.353052020072937, 2.337608814239502, 1.100825548171997, -0.18268240988254547, -0.16378694772720337, 0.4060560464859009, -2.3558926582336426, -1.5721946954727173], [0.21860529482364655, 1.294670820236206, 1.7935564517974854, 0.4595530927181244, 1.5253901481628418, 0.12211234867572784, 0.5557069182395935, -0.36668288707733154, -1.405814290046692, -0.8357160687446594, 0.6345207095146179, -0.5779303908348083, -1.4407018423080444, 0.24924376606941223, 2.6847033500671387, 1.634706735610962, -1.5375151634216309, 1.292893409729004, 1.348042368888855, -0.03514249250292778, 0.8104054927825928, 0.6978165507316589, 1.0887322425842285, -0.17127588391304016, -0.14276906847953796, -1.0657620429992676, -0.22972320020198822, 0.6872656345367432, 0.6928166747093201, 0.059498682618141174, 0.02373761683702469, -0.043640196323394775, 0.8530226349830627, -0.8121101260185242, -0.5512983202934265, -1.1079137325286865, -0.9658628702163696, 1.7665257453918457, -0.43132254481315613, 1.1532530784606934, -1.5915825366973877, 0.02357509918510914, 0.015646012499928474, 1.1336650848388672, -0.8343954682350159, -0.3538512885570526, 0.23571287095546722, -0.5896095037460327, 1.6503719091415405, -1.2182036638259888, 1.366613507270813, -1.805932641029358, 2.755619764328003, -0.12843570113182068, 0.9187455773353577, 0.13441777229309082, 0.5017005801200867, 0.12639328837394714, 0.7767465114593506, -1.3048646450042725, -0.06508409231901169, -0.3858901560306549, 0.4969216287136078, -2.1017048358917236], [-0.5584258437156677, -0.6280712485313416, 1.7449487447738647, 1.1092127561569214, 0.8348028659820557, 0.21892404556274414, 0.17392221093177795, -1.2466033697128296, -0.9749394655227661, 0.6020357012748718, 0.8108869194984436, -0.8538556694984436, 0.831892728805542, -0.8950169086456299, 0.26600778102874756, -0.3237507939338684, 0.6455522775650024, 0.043322958052158356, -0.2996077835559845, 0.5343952775001526, 0.06404734402894974, 0.453594446182251, -0.27835556864738464, 2.454411745071411, -1.2003742456436157, 0.9886402487754822, -0.2810100317001343, 1.1159319877624512, -0.32442083954811096, 0.9258794784545898, 2.6694304943084717, -0.9051360487937927, -1.3189738988876343, 1.4397461414337158, -0.4895765781402588, -2.1216297149658203, -0.21184702217578888, -1.3884857892990112, -1.090085744857788, 0.6258749961853027, -0.035761479288339615, 0.8246563673019409, -0.29264700412750244, 0.17265209555625916, -0.4870496094226837, 0.4633949100971222, 1.2396982908248901, -0.3398243486881256, 2.372955799102783, -0.6897061467170715, -1.7849056720733643, -0.654575526714325, 1.8513530492782593, -0.19403067231178284, 0.4848315119743347, 2.097362995147705, 0.42836859822273254, 0.04190254211425781, -1.173960566520691, -1.3529947996139526, -0.016448305919766426, 1.1547645330429077, -1.4383518695831299, -0.27889400720596313], [-1.500440239906311, 1.052250623703003, -0.027472687885165215, -0.714005708694458, -0.27775201201438904, 2.2283308506011963, 0.08555468171834946, 2.673696279525757, -0.1659928262233734, -0.7391124963760376, 0.2966899275779724, 1.8984678983688354, -0.3331044316291809, 0.23505660891532898, 2.1185567378997803, 1.3951925039291382, 0.6042458415031433, -0.2299034148454666, -1.6872326135635376, -0.5239519476890564, 0.3973168432712555, -0.40602046251296997, 0.17542439699172974, -0.9425041675567627, -0.05401301383972168, -0.12780453264713287, 1.2226272821426392, 0.3629244863986969, 0.36897799372673035, -0.2512767016887665, 0.13207247853279114, 0.5541733503341675, 0.19708584249019623, -0.046521902084350586, 0.5842717885971069, 0.4957689642906189, 0.8650994896888733, 0.06105181202292442, -1.2294855117797852, 0.2294754534959793, -0.2106582522392273, 1.1137912273406982, -1.0727664232254028, -0.5790515542030334, 0.1538182646036148, 1.116265058517456, 0.2704755961894989, 0.5311604738235474, 0.4431697428226471, 0.4406416416168213, 1.8105542659759521, 0.2080727070569992, -0.6141895651817322, 0.558457612991333, 0.3727478086948395, 0.006733720190823078, -0.7478307485580444, 1.627138614654541, 0.06353633105754852, -1.63107168674469, -0.4739901125431061, -1.3001041412353516, -1.1140085458755493, -0.49231410026550293], [1.3768837451934814, 0.09882281720638275, 1.2859530448913574, -0.3612605035305023, -1.2282301187515259, 0.9617994427680969, 1.6982532739639282, 0.01001308299601078, -1.1401745080947876, 0.8516960144042969, -1.0769305229187012, -0.2703292965888977, 2.0172910690307617, 1.670407772064209, 0.15178938210010529, 0.3110356032848358, -0.020504871383309364, 0.8337008357048035, 0.9870441555976868, -0.40664443373680115, 0.5729527473449707, 1.3514623641967773, 0.4547847509384155, -0.531658411026001, 1.0399391651153564, 1.380882978439331, 0.10910982638597488, 1.218248963356018, -0.6479478478431702, 0.7668328881263733, 0.7346134781837463, -1.7169047594070435, -0.5094572305679321, 0.5825349688529968, 0.10688353329896927, -0.14163753390312195, -1.6828101873397827, 0.9559517502784729, 0.8023573756217957, -1.7258431911468506, -2.1093387603759766, -0.1386052817106247, -0.10965678840875626, -0.4553765058517456, 0.189395010471344, -1.50806725025177, 0.1764560341835022, -0.6853334903717041, 0.08075812458992004, 0.003398864297196269, 1.1190191507339478, -0.22150374948978424, 0.4025951027870178, 1.770532488822937, 1.4088866710662842, 0.14061643183231354, -0.4684417247772217, -1.500036358833313, 0.23704794049263, -0.940963089466095, -0.9402354955673218, -0.9104337692260742, -0.49971726536750793, -0.5227342844009399], [0.18191266059875488, -0.21106921136379242, 1.0503309965133667, -1.2475810050964355, -0.48578429222106934, -0.16410788893699646, -0.0806041955947876, -0.049258239567279816, 0.29427430033683777, -0.24506445229053497, 0.1492432951927185, 1.8550689220428467, -1.0836601257324219, 0.24131999909877777, -0.10100027173757553, -0.21340182423591614, -0.7914363741874695, -1.3214490413665771, 2.1622352600097656, -0.7098560333251953, 0.37237247824668884, -0.3077773451805115, -0.4288070499897003, 0.7096017599105835, -1.1800556182861328, 0.5154109597206116, -0.6091048717498779, 0.12328164279460907, 1.9343916177749634, -0.0008725334191694856, -1.3183355331420898, -1.2830191850662231, 0.8641238808631897, 0.32158249616622925, 1.0246614217758179, 0.2073606550693512, 0.08407779783010483, -0.7524262070655823, -1.1675668954849243, 0.29821211099624634, 0.6378746628761292, -0.003945319913327694, 0.5901219844818115, -0.9827567934989929, -1.127629041671753, 1.71513032913208, 0.9394354224205017, -0.12323293089866638, 0.22206470370292664, -0.6744037866592407, 1.2477422952651978, -0.29218029975891113, -0.30580076575279236, -0.4756348133087158, -0.6138166189193726, 0.028973689302802086, 0.01564689539372921, -0.19719842076301575, 0.37578627467155457, 0.38440632820129395, -0.2030658721923828, 0.05218358710408211, -0.5939542055130005, 0.48600512742996216], [0.24865153431892395, 1.7413270473480225, 0.34508419036865234, -1.021872639656067, 0.5810531377792358, 0.10200836509466171, -0.3017948567867279, 2.400177001953125, 0.9265992641448975, -0.7282293438911438, 1.6959800720214844, 0.27557694911956787, 0.42530304193496704, -1.5694122314453125, 0.7692527770996094, -0.003852797206491232, -0.4372217357158661, 1.128618836402893, -1.0123820304870605, 0.48110368847846985, 0.34016016125679016, -1.0069066286087036, 1.2656362056732178, -0.40786683559417725, -0.5064010620117188, 0.1328715682029724, -0.35833659768104553, -0.9859869480133057, -0.341802716255188, -0.28920021653175354, -1.2996082305908203, -0.12369677424430847, 0.923089861869812, 1.1258658170700073, 0.5171814560890198, 0.651850163936615, 0.7999390959739685, 0.09892889112234116, 1.4385898113250732, -0.4936028718948364, -0.5689167380332947, 0.20005613565444946, 0.7159045338630676, -0.785685122013092, -0.14419785141944885, 0.9565882086753845, 1.0417159795761108, 1.3774231672286987, 1.4438204765319824, 0.16782814264297485, -0.7852239012718201, 0.27899783849716187, 0.9586000442504883, -1.6538934707641602, -2.0375709533691406, 1.3188822269439697, -0.5543436408042908, 0.1612129956483841, 0.7095350027084351, 0.35850071907043457, 0.5037276148796082, -0.8662235736846924, 1.5377860069274902, 1.862043023109436], [0.016113029792904854, 0.7864935398101807, -1.9360407590866089, 1.1230173110961914, 0.5652502775192261, 0.15276020765304565, 0.9289658069610596, 0.8224817514419556, -0.34406930208206177, 0.19672206044197083, -0.48859167098999023, -0.18369445204734802, -0.9120064377784729, 0.028488177806138992, 0.144779235124588, -1.9249638319015503, -0.7107139825820923, 1.332243800163269, 0.3704599142074585, -0.6394636034965515, -1.4019482135772705, 0.5645148158073425, 0.40701404213905334, -0.5297476053237915, 0.09533893316984177, 1.3380683660507202, -1.6590282917022705, -0.32666224241256714, -0.758118212223053, -0.04174191504716873, -1.2488648891448975, -1.1994190216064453, -0.773511528968811, 0.6823253631591797, -0.3874795436859131, -0.052486155182123184, -0.5023442506790161, -0.5130866169929504, -0.7735939025878906, 1.2256925106048584, 1.1993080377578735, 2.1111953258514404, 0.2797226905822754, 0.060717545449733734, 0.32018935680389404, 0.6789566874504089, -0.12899336218833923, -1.4185073375701904, 0.7009814977645874, -0.752640962600708, -0.975640058517456, 0.5575070977210999, -0.39833423495292664, 1.461382269859314, 0.46403926610946655, 0.46276187896728516, 1.4340267181396484, 0.7144790887832642, -1.2131240367889404, -1.8002341985702515, 0.6994110345840454, -0.35295379161834717, -0.5183676481246948, -0.7957909107208252], [-0.36101672053337097, -2.404808521270752, 0.02121567539870739, -0.29357030987739563, 0.6972755193710327, -1.0067499876022339, -1.4017465114593506, -0.40061259269714355, 0.9715200662612915, -0.5344617962837219, 0.5642703771591187, 1.5807033777236938, 0.6925009489059448, -0.943081259727478, -0.2759508490562439, -0.23222333192825317, -0.014378504827618599, -0.2922587990760803, -2.555047035217285, 0.5109870433807373, -1.400836706161499, 0.8026031255722046, -1.2541441917419434, -0.19932043552398682, 0.5590956211090088, 1.4807548522949219, 0.2612275779247284, 0.08521240204572678, 0.3176702558994293, -1.0175329446792603, -0.43776950240135193, 0.8085232377052307, 0.6348850727081299, -0.5907103419303894, -1.181140661239624, -0.7693197727203369, -0.917129397392273, 1.1994738578796387, 0.311284601688385, 2.2578189373016357, 0.8946730494499207, 0.4350021481513977, 1.0495080947875977, 0.8705738186836243, 1.2660590410232544, -0.21471837162971497, -1.0238101482391357, 0.4274204969406128, 0.051464978605508804, -0.29052650928497314, 0.7967418432235718, 1.7938740253448486, 0.6623108386993408, 0.5443024039268494, 0.12987682223320007, -0.49971142411231995, 1.321210503578186, 0.13648267090320587, -0.7309901714324951, 0.0023333861026912928, 1.5768083333969116, 0.3688845932483673, -0.20497570931911469, -0.5496179461479187], [1.0368194580078125, 0.42792537808418274, 0.09340299665927887, -0.7042489647865295, 0.10425468534231186, 0.5097887516021729, -0.2744397819042206, 0.7180579304695129, 0.6992683410644531, -1.9857816696166992, 1.4140342473983765, -1.0161200761795044, 0.13160055875778198, -1.061594009399414, -2.4547369480133057, 0.6341308355331421, -0.6592295169830322, 0.6281509399414062, -0.873860239982605, -0.1492534577846527, 1.74795663356781, 0.3535999059677124, -1.2502604722976685, 0.9077931046485901, -1.1833693981170654, -0.6885464787483215, -1.3651841878890991, 0.7906712293624878, -0.679135799407959, -0.4038558304309845, -0.8654788732528687, -1.9051384925842285, 0.8360745906829834, 0.9352989792823792, 0.1862390786409378, 0.4079729914665222, 1.5374099016189575, -0.7949997782707214, -0.7681766748428345, -0.2259558141231537, -0.12541598081588745, 1.180289387702942, -0.8673226833343506, -0.40705201029777527, -1.1390326023101807, 0.7509545087814331, 2.1299657821655273, 0.03046959824860096, -1.44932222366333, 1.7591081857681274, 1.149670958518982, 0.7978695631027222, -1.0122631788253784, 1.2337757349014282, 0.3233235776424408, -0.2584822177886963, -0.3247636556625366, 2.3169236183166504, -0.6009141802787781, 0.03226647898554802, -1.077439546585083, -1.0146335363388062, -0.8007129430770874, 0.5067716240882874], [-0.2540320158004761, -0.5107738971710205, -0.33891987800598145, 0.2969205677509308, -0.567510724067688, 0.5125366449356079, 1.4358748197555542, 0.28383544087409973, 0.3883918225765228, -0.9266955256462097, -0.8616152405738831, 0.765824019908905, 0.10517792403697968, 0.1272164136171341, 0.47338661551475525, -0.8596817255020142, 0.11246059834957123, -0.9084906578063965, 1.1578551530838013, -0.05884512513875961, 0.11806890368461609, 0.8199263215065002, -0.280163437128067, 1.8964080810546875, -0.42779117822647095, 0.33949917554855347, -0.2843874990940094, -0.1267504096031189, -0.2980055510997772, 1.5205955505371094, -0.8139581680297852, -0.49466609954833984, -0.9328704476356506, 0.29901161789894104, 1.7236279249191284, 0.06423439085483551, -0.35134217143058777, 1.7174283266067505, 1.9227888584136963, -0.7749377489089966, -0.7287750840187073, -1.7355362176895142, 0.656193196773529, 2.2750158309936523, -0.5489107966423035, 0.39874252676963806, 0.9186110496520996, -0.292520135641098, 1.5091356039047241, 0.30790552496910095, 1.6174858808517456, -1.728888988494873, -0.37292933464050293, 1.441009283065796, -0.08097710460424423, -0.49463415145874023, -1.6706771850585938, -1.1641185283660889, -0.679137647151947, 0.3502865135669708, -0.36736345291137695, 1.2835177183151245, -1.2679240703582764, -0.21474607288837433], [0.41022664308547974, -0.7286906242370605, 0.21146121621131897, -0.26452985405921936, -0.16976326704025269, -0.4889998435974121, -0.5438706278800964, -1.017903208732605, 1.1401796340942383, 0.1788109689950943, -0.9112181663513184, -0.3252708613872528, -0.38741806149482727, -0.11295055598020554, 0.7788584232330322, 2.417418956756592, 0.3607836961746216, 0.7801625728607178, 1.4554532766342163, 1.1305896043777466, 0.8225911259651184, -0.042120322585105896, -0.589830756187439, 2.1134188175201416, -0.019955145195126534, -0.5040011405944824, 1.6146695613861084, -1.104091763496399, -0.7637442946434021, 0.4320387542247772, 0.6008292436599731, -1.9740396738052368, 0.6895055770874023, 0.19485321640968323, -0.9982305765151978, -0.030567586421966553, 2.5498266220092773, 0.6909704208374023, -0.23447322845458984, -1.3913052082061768, 1.3844528198242188, 0.7291591167449951, 0.4652974307537079, 1.262664794921875, 0.8435492515563965, 0.8198339939117432, -1.1799885034561157, 0.8404229879379272, -0.32122939825057983, -2.3236782550811768, -1.965484619140625, -0.9096238017082214, 0.9735767245292664, 2.163379192352295, -1.1468044519424438, -1.776357650756836, 0.5756712555885315, -1.6277300119400024, 0.6561123132705688, -0.40336406230926514, 0.38239771127700806, 1.2617871761322021, -0.5580965876579285, 0.9896392226219177], [-1.0593854188919067, -0.68567955493927, -1.6376854181289673, -1.0052804946899414, 1.8267039060592651, 0.0452692024409771, -0.8976036310195923, 1.0611426830291748, 0.8189363479614258, 0.9555003046989441, 0.07153719663619995, -0.23373441398143768, 0.10738686472177505, 0.6091178059577942, -2.136347532272339, -1.161346435546875, 0.28325995802879333, 0.10535168647766113, -0.28215110301971436, 1.0611664056777954, 0.28867480158805847, -1.8224238157272339, 0.08356348425149918, 1.3227226734161377, 0.12014767527580261, 0.7110580801963806, 0.41336411237716675, 0.39399996399879456, -0.704393208026886, 1.9764906167984009, 0.2544139325618744, -0.9684781432151794, -0.012541188858449459, -1.0313889980316162, 0.9473108649253845, -0.8308702111244202, -0.3273015022277832, -0.813513457775116, -0.1962268352508545, -1.4531794786453247, -0.6582670211791992, -1.03826904296875, -0.48433414101600647, -1.2064859867095947, -1.7721781730651855, -0.6749807000160217, 0.8717024326324463, 0.26748576760292053, 2.2319345474243164, 0.3219696283340454, 0.803540825843811, 0.5937315821647644, -1.1376678943634033, -1.273202896118164, 0.7701507210731506, -0.2376057356595993, 1.3600797653198242, 0.6140503883361816, -0.33936047554016113, -0.5485855340957642, 0.44943174719810486, 1.1987755298614502, -0.020879041403532028, 1.7918307781219482]], [[-0.05032194405794144, 0.6139934659004211, -0.23868051171302795, 2.523383855819702, -1.9692003726959229, 0.1435449719429016, 0.2543344795703888, -1.1029877662658691, 1.2948834896087646, 0.46238139271736145, -0.4028625786304474, 1.1357922554016113, 1.7000876665115356, -0.005988330114632845, -1.1237859725952148, 0.2871800661087036, -0.48134729266166687, -0.820238471031189, 0.3639642596244812, -0.9464924931526184, -0.4401889443397522, -0.6861838102340698, -2.549450635910034, 0.5890542268753052, 0.46598368883132935, 1.4877076148986816, 0.8603625893592834, 0.5234678983688354, 1.411971926689148, -0.012995832599699497, -0.8506551384925842, -1.6067304611206055, 0.19141660630702972, -0.15698695182800293, -0.6407449841499329, 0.45397406816482544, 1.116400957107544, -1.557573914527893, -0.3569810390472412, -0.8119767308235168, -0.6583108305931091, 0.22017648816108704, 1.4277039766311646, -0.35621902346611023, 0.39081329107284546, 0.5853956937789917, -1.2213822603225708, 0.4037838578224182, -1.4302856922149658, 1.4405841827392578, 0.37119653820991516, 0.7672760486602783, 2.598022937774658, -0.12495653331279755, -1.185691237449646, -0.4659308195114136, 1.63821280002594, 0.6790518164634705, -0.05349921062588692, 2.857250452041626, 0.3786410689353943, 2.2057456970214844, 0.506767988204956, -1.090197205543518], [0.5623497366905212, 0.06257578730583191, -0.5678114295005798, -0.8482515215873718, -1.4899927377700806, -0.08928105235099792, -0.5565006136894226, -0.0492573156952858, 0.2656250596046448, 0.46544837951660156, 0.13529561460018158, 1.0828120708465576, -0.5152488946914673, 1.8166799545288086, 0.38586392998695374, 1.1942403316497803, -1.6697988510131836, 0.450723260641098, 2.236339569091797, 0.5284872651100159, -0.8699309229850769, -0.09814759343862534, -1.0799225568771362, -0.6702656149864197, -0.5960955619812012, -1.5042567253112793, 0.21825581789016724, -0.44191983342170715, 0.5672245621681213, 0.5713778734207153, 1.1344759464263916, 0.15094581246376038, 0.250661164522171, 0.04426807910203934, 2.3729453086853027, -1.0713257789611816, -1.6737059354782104, -0.5812190771102905, 0.3259201943874359, 0.06235616281628609, -0.3211696445941925, 0.6071818470954895, -1.1054290533065796, -1.470836877822876, -1.463175892829895, 1.5945584774017334, 0.14383834600448608, 0.10218512266874313, -1.3440488576889038, 0.8555316925048828, 1.0731728076934814, -0.16107764840126038, -0.6071434020996094, 1.4534235000610352, 1.0362776517868042, 1.784949779510498, 1.0070573091506958, 0.4021152853965759, 1.76524817943573, 0.2676483988761902, -0.5393209457397461, 1.6516741514205933, -0.16099779307842255, 0.8415421843528748], [1.5267927646636963, -0.6496158838272095, -0.4437173306941986, 0.5099347829818726, 0.5773559808731079, -0.9755963087081909, -0.3736696243286133, 0.09003733098506927, 0.35694459080696106, -0.26087242364883423, -0.3256400227546692, -1.4300899505615234, -1.4610087871551514, -0.995805025100708, -0.25363337993621826, -0.5662125945091248, 0.4670659303665161, -0.08204777538776398, 0.6808890700340271, -1.6838645935058594, -0.5669540166854858, -1.5096358060836792, 0.16963574290275574, 0.4220358431339264, 1.8801774978637695, 1.0459480285644531, 2.1228504180908203, -0.3594316244125366, -2.4395086765289307, 0.23665562272071838, -1.2758288383483887, 0.789622962474823, 1.3259435892105103, 0.20116764307022095, -0.522826611995697, 0.6133145689964294, -1.0780978202819824, -1.1930640935897827, -0.2000173032283783, 1.4341658353805542, 0.9574041962623596, 0.2512364089488983, 0.8930121064186096, -1.2718491554260254, 0.6822529435157776, -1.5237590074539185, -0.5666240453720093, 0.31001561880111694, 0.41453617811203003, 0.7000404596328735, 0.2930688261985779, -2.0182533264160156, 0.8933807015419006, -0.3056323826313019, -0.38434576988220215, 0.6710090637207031, -0.2196219265460968, -1.3272514343261719, 0.8395134210586548, -0.009404756128787994, -2.2670013904571533, 0.8185943365097046, -0.6723504066467285, 0.4836233854293823], [-1.5903394222259521, 1.7029894590377808, -0.9096994400024414, 0.08620722591876984, -1.1338791847229004, 3.0389111042022705, 1.1603294610977173, 0.665594220161438, 0.5402476191520691, -4.009869575500488, 0.7748368382453918, -0.9893496632575989, -0.3867564797401428, 0.161997452378273, 0.2628032863140106, 0.8509207367897034, -0.5988951921463013, -0.04408811032772064, -0.572694718837738, -0.39455440640449524, 1.165468454360962, 0.11579783260822296, 0.48275789618492126, -0.7565105557441711, 0.25840240716934204, 0.5366093516349792, 0.44966796040534973, 0.9166591763496399, 1.4412721395492554, 0.5466428399085999, -1.668561577796936, -0.05801486223936081, -0.6091936826705933, 0.38937222957611084, 0.7016289830207825, 1.3240957260131836, -0.6368206143379211, -0.7535066604614258, -1.0903046131134033, -0.7842862606048584, -0.8045253753662109, 0.9748955368995667, -1.410884141921997, -0.17973269522190094, -0.28792738914489746, -1.6575335264205933, -0.08199361711740494, -0.6646639108657837, -0.774258017539978, -0.5054036974906921, 0.6980558037757874, -0.45594754815101624, 0.2903386354446411, 0.4125041961669922, -0.2835600972175598, 0.38221392035484314, -0.10973434895277023, 1.0656815767288208, -0.2308972179889679, -0.08695570379495621, 0.323961079120636, 0.006597969215363264, -0.09539350122213364, 1.5765268802642822], [-0.1838994175195694, -1.1510480642318726, 0.8250682353973389, 1.052981972694397, -0.7767958045005798, -1.5440378189086914, -0.3313240110874176, 0.2883780598640442, 0.8313236832618713, -1.8371727466583252, 0.7593984007835388, 0.906252384185791, 1.6760927438735962, -0.36582037806510925, 0.028947288170456886, -1.0547879934310913, 2.4980080127716064, -0.3884139955043793, 0.25746950507164, 0.4405912756919861, -2.2834179401397705, -1.2506333589553833, 1.5536516904830933, -1.3697550296783447, -0.5615829825401306, -1.754393219947815, -1.810571551322937, 1.4253182411193848, 1.167500615119934, -0.8523988127708435, 0.3036983013153076, -0.3824409246444702, 1.200542688369751, -0.223196879029274, -0.5564466118812561, -1.3255301713943481, 0.01822555810213089, -0.8003758788108826, -0.18609589338302612, 0.5009996294975281, 1.0624680519104004, 0.400812566280365, -1.294636607170105, -2.2787208557128906, -1.18973970413208, 0.9201677441596985, 1.5378050804138184, -0.41924527287483215, -0.880224347114563, 1.2584599256515503, 0.15078113973140717, -0.33805903792381287, 0.40044882893562317, 1.340358853340149, -0.24918220937252045, -0.8095788955688477, 0.6758635640144348, -1.3020856380462646, -0.15856905281543732, 0.6022634506225586, 1.479243516921997, -1.0754783153533936, -0.8397020697593689, 0.7046645879745483], [-0.45450448989868164, -0.16039493680000305, -0.095054991543293, 0.9844350218772888, -1.9109941720962524, 0.5681086182594299, -0.6890646815299988, 0.10427597165107727, 0.2883285582065582, -1.3233520984649658, 0.17554804682731628, 0.5137053728103638, 1.5746444463729858, -0.4368470311164856, -0.24276532232761383, 0.05096418783068657, 1.6506460905075073, 1.630861520767212, -0.7386249303817749, -1.9955915212631226, -0.44527214765548706, 1.4117043018341064, -0.22905568778514862, -1.018355369567871, 0.5014851689338684, -0.7204133868217468, 0.5272627472877502, 1.4462329149246216, -0.4237446188926697, -0.1917094588279724, -1.0685911178588867, 0.060108501464128494, 1.7435303926467896, 1.1603891849517822, 0.7726275324821472, -0.29563459753990173, -0.7389194369316101, 0.02260305918753147, 0.21092116832733154, -0.956707775592804, -0.3788515627384186, 0.6698975563049316, 1.4148610830307007, -2.315169095993042, -0.5590934157371521, -0.06249712407588959, -0.5053567886352539, -0.9507376551628113, -0.604608952999115, 1.7173771858215332, 0.5456147193908691, -1.306547999382019, -0.19924409687519073, 1.987121820449829, 0.8079807758331299, -1.058716058731079, 0.0036232839338481426, -0.43971195816993713, 0.07832443714141846, -0.9640861749649048, 0.3849652409553528, 1.3412537574768066, -0.05717958137392998, 0.12563243508338928], [0.7549288272857666, -0.5388621687889099, -1.5879031419754028, -1.1830912828445435, 0.8412424325942993, 0.44811657071113586, 1.071799397468567, 0.6251472234725952, -0.0387142151594162, -0.3153130114078522, -0.34322455525398254, -0.23970189690589905, -1.1852130889892578, -1.5310955047607422, -1.3370285034179688, -0.2739446759223938, -0.8980713486671448, -0.5635215640068054, -0.5744501948356628, -0.7365420460700989, -1.3556358814239502, 0.8951015472412109, -1.6176578998565674, -0.3160950541496277, -0.6073072552680969, 0.3890746235847473, 0.9995114803314209, -0.9410274624824524, 0.8087924122810364, -1.0109103918075562, -0.8853089213371277, -0.4117686152458191, -1.204655408859253, -0.28257662057876587, 0.19086405634880066, 1.039374828338623, 0.9050760269165039, 1.1560308933258057, -1.1603890657424927, 0.46062400937080383, 0.3181462585926056, -0.4820282459259033, 2.662978410720825, -1.8642215728759766, 0.11057452857494354, -1.3162615299224854, -0.03879069164395332, 0.5129857063293457, 0.2447476089000702, 0.8669275045394897, 0.6228206753730774, 0.7997635006904602, -0.3042772114276886, -0.39318785071372986, -0.41228315234184265, -0.7945440411567688, -2.302886962890625, 0.5917810797691345, -0.3392362892627716, 1.7022323608398438, 0.06558477878570557, 0.20410284399986267, -1.4421257972717285, 0.15870238840579987], [1.1324599981307983, -0.532013475894928, 0.4639238119125366, -0.10458142310380936, 0.10479670017957687, -0.26910319924354553, -1.2524768114089966, -0.6077625751495361, 0.7913779020309448, -0.8178504109382629, 1.1359254121780396, -0.7899373173713684, -1.1559451818466187, 0.9687789082527161, 1.3785392045974731, 0.007240555714815855, 1.332907795906067, 0.6121354699134827, 1.6068378686904907, -1.704201340675354, -1.8911688327789307, 0.24531099200248718, -1.266616702079773, 0.993873119354248, -2.351719379425049, 0.25971633195877075, 0.950161874294281, -1.0229315757751465, -0.16115717589855194, -0.07413479685783386, -0.5152167677879333, 0.3810807466506958, 0.29930418729782104, -0.9799110889434814, -0.14835035800933838, -0.5864117741584778, -2.9209070205688477, 1.2903249263763428, 0.887884795665741, -1.034501552581787, 2.972668170928955, -0.9715632200241089, 0.8049347400665283, -1.1144750118255615, -0.4561734199523926, 0.5645124316215515, 0.2672058343887329, -2.084841012954712, 0.29822903871536255, 1.9902853965759277, 0.755501389503479, 0.9066084623336792, -0.6369521021842957, 0.03870975971221924, -1.6198474168777466, 0.5027350783348083, -1.4993168115615845, 1.4653387069702148, -0.40709465742111206, 1.7880405187606812, -1.2349756956100464, -0.3652817904949188, -0.038364339619874954, -1.2031856775283813], [2.032392978668213, 0.6354747414588928, 1.2317049503326416, 0.9921294450759888, -0.36327236890792847, 1.3023813962936401, -0.508516252040863, -0.8277432322502136, 0.296862930059433, 0.2582179307937622, -0.9173373579978943, -0.089591845870018, 3.220625162124634, 0.14123617112636566, 0.8894318342208862, -0.8738709092140198, 0.11722008883953094, -0.14073322713375092, 0.602075457572937, -0.32451456785202026, -0.07649946212768555, -0.022134121507406235, 1.7436630725860596, 0.9705146551132202, -0.20746533572673798, 0.6989296078681946, -0.4262044131755829, -1.242477536201477, 0.42945703864097595, 0.18536600470542908, -0.18071700632572174, -0.8416637182235718, -1.1728541851043701, 0.1939162313938141, 0.46102362871170044, 0.3610829710960388, 0.39834171533584595, -0.3719608187675476, -0.344663143157959, -0.716894805431366, -0.6788580417633057, -1.338962197303772, -0.502403736114502, 0.4895433783531189, -1.656043529510498, 0.5361753106117249, 0.13550472259521484, -0.4355325996875763, -1.9370359182357788, -0.6774973273277283, 0.9586169719696045, -1.1814243793487549, -0.5001108646392822, -1.3301135301589966, 0.6942260265350342, -0.031177416443824768, -0.2853785753250122, 0.40390992164611816, 0.03313208371400833, 1.361219048500061, -0.31205639243125916, 0.20567163825035095, -0.09312168508768082, -2.306030035018921], [0.03652653843164444, -0.37738025188446045, -0.7891831398010254, 0.23848196864128113, -0.08184440433979034, -1.3187048435211182, -0.26657766103744507, -0.49549704790115356, -0.0025524974334985018, 0.7547307014465332, -0.24657954275608063, 0.016358118504285812, 0.7469179630279541, -1.3660368919372559, -0.8576256632804871, -1.982079267501831, 2.079982042312622, -1.2094379663467407, -0.2924533486366272, 0.43223294615745544, -0.3735100030899048, -0.8076034188270569, -1.1471400260925293, -0.41143471002578735, -0.00553514901548624, -1.2943562269210815, -0.6760528683662415, 2.0691981315612793, 0.2447412759065628, 2.041032314300537, 0.8867286443710327, -1.0852025747299194, 1.3607375621795654, -0.11382018029689789, 0.23732879757881165, 0.027693066745996475, -1.4662317037582397, -1.5939682722091675, 1.0332859754562378, 0.15544386208057404, -0.22679798305034637, 1.0980712175369263, -0.65547776222229, 0.6435108184814453, 2.317047357559204, -1.9848805665969849, 2.016146183013916, -0.3077251613140106, -0.47822996973991394, 1.2623533010482788, -0.37924525141716003, 0.40325963497161865, -0.5342373251914978, 0.20040598511695862, 2.4253785610198975, -0.18493451178073883, -1.4119046926498413, -0.46800416707992554, 0.1273290514945984, -0.7988114356994629, -0.07848253101110458, -1.6837903261184692, -1.286149024963379, -1.3330953121185303], [0.05122704803943634, 0.2786497473716736, 0.2931724190711975, -0.22711458802223206, 0.5333088040351868, -0.9660214781761169, -0.9857478737831116, -0.7315292954444885, 0.3481466472148895, -0.1498592346906662, 1.7701586484909058, -0.4046269655227661, 0.2808101773262024, -0.43796178698539734, -0.5025380849838257, -1.0463932752609253, 0.10963068157434464, 0.11644898355007172, -0.07677394896745682, 0.2523408532142639, -0.10422442108392715, -0.4415571689605713, 0.901329517364502, 1.302785873413086, -1.4367151260375977, -0.21727201342582703, -0.16252680122852325, 0.4872787892818451, 0.5578277111053467, 0.5915714502334595, 1.5488187074661255, -0.1180860698223114, 0.6009745001792908, 0.12091991305351257, -2.4298436641693115, -0.8018500208854675, 1.5306570529937744, -0.0833669900894165, 1.3190206289291382, -0.4429263770580292, 0.874699592590332, -0.15870891511440277, 0.8372701406478882, -0.5976212620735168, 0.5323631167411804, -1.1740739345550537, 0.21339361369609833, 0.404167115688324, -1.137555480003357, 1.7118103504180908, -0.554524302482605, 0.09565021097660065, 1.992106318473816, -0.764118492603302, 0.23841580748558044, -0.7282887101173401, 0.19394633173942566, -0.07182197272777557, -0.3476918339729309, -1.9167569875717163, -0.03673470765352249, -0.5485442280769348, 0.3229328691959381, -0.54071044921875], [-0.6599725484848022, -0.3037644326686859, -0.3467783033847809, -1.167272686958313, 1.9013456106185913, 0.6527412533760071, -1.8585273027420044, -1.5771644115447998, 1.03816819190979, 0.588490903377533, -0.006065999157726765, 0.4751916527748108, 1.5200543403625488, -0.3576442003250122, 2.091392755508423, 1.458184003829956, 1.3121552467346191, 0.3449932634830475, 0.022438276559114456, 0.10596717149019241, 0.26848849654197693, 0.5693750381469727, -0.36579039692878723, -0.5668138265609741, 1.8888288736343384, 1.1766501665115356, 0.5238719582557678, -0.5643908977508545, 0.8030316233634949, -0.7135840654373169, 0.748611569404602, -0.10824970155954361, -1.0444350242614746, 0.37783777713775635, -0.745919942855835, -1.1951326131820679, 1.468966007232666, 0.7626032829284668, 0.7249035835266113, -0.36458200216293335, 1.516045331954956, 0.5747963786125183, -0.629179060459137, 0.9130244255065918, -1.1677227020263672, -0.14339075982570648, 1.2139770984649658, -0.6950072050094604, -1.9501962661743164, 0.542930006980896, -0.7917932271957397, -1.349891185760498, -0.2840479016304016, 1.0352295637130737, 0.6735486388206482, 1.2557995319366455, -0.5235954523086548, -0.8543421626091003, -0.6671690344810486, 1.1196507215499878, -1.2905746698379517, 0.9422374367713928, 1.0978070497512817, -0.9713641405105591], [-0.5139659643173218, 0.6906929612159729, 1.2386126518249512, -0.03710222616791725, -0.3000011146068573, -0.052269939333200455, 1.953505277633667, 0.4590606689453125, 0.9812326431274414, 0.7503272891044617, 0.016946088522672653, -0.9597927331924438, -0.10020221024751663, -0.5706336498260498, -0.5797598361968994, 0.7848669290542603, -0.10091566294431686, -2.251840829849243, -0.621372640132904, 0.46075019240379333, 1.7284152507781982, -0.15609878301620483, -0.873856782913208, -1.4743421077728271, -0.7659716606140137, -0.20279447734355927, 1.1535253524780273, -0.36666861176490784, 1.2553989887237549, 1.4333748817443848, -1.5310277938842773, 1.0287299156188965, 1.3671743869781494, 0.2157285511493683, 0.468569278717041, -0.8929482102394104, 0.2575056850910187, -0.06658212840557098, 0.5069475173950195, 1.0169097185134888, 1.7020225524902344, 0.48130643367767334, 0.0044777896255254745, 1.8328782320022583, 1.8200567960739136, 0.6633699536323547, 1.261907696723938, 0.29469746351242065, -0.7019959688186646, 1.1237716674804688, 0.12959565222263336, -1.1617926359176636, -1.5800230503082275, -0.41964924335479736, 1.8738189935684204, -0.7749502658843994, -0.327880859375, -2.8690097332000732, -0.16402190923690796, 0.11378935724496841, -0.7964664697647095, -0.22323957085609436, -0.06471472978591919, 0.4508940279483795], [0.6107978224754333, -1.811559796333313, 1.285079836845398, -0.18175257742404938, 2.7832350730895996, 1.7132346630096436, -0.31668540835380554, -0.505645751953125, -1.4550747871398926, 1.543052077293396, -1.8114778995513916, 0.04378893971443176, 0.7756555080413818, 0.13698671758174896, -0.4364147186279297, -0.1664378046989441, -0.7591180205345154, -0.4139571785926819, 0.6071743369102478, 0.6739818453788757, 0.48674896359443665, 0.48240429162979126, 0.47209325432777405, 0.30740708112716675, 0.30380213260650635, 0.44008398056030273, 0.6751828789710999, -0.3980139195919037, -1.017146110534668, -1.2872099876403809, 1.3582149744033813, -0.5559400916099548, -1.3532651662826538, -1.2807759046554565, 0.10172668844461441, 0.4462323486804962, -0.4863033890724182, -0.6627179980278015, -0.26527804136276245, 0.09781096130609512, 1.1214088201522827, 0.8693507313728333, -1.1590282917022705, -1.3267570734024048, -1.3207595348358154, 0.7432542443275452, -1.221901535987854, 0.9352620840072632, 0.10350649803876877, -0.40715664625167847, 0.49179714918136597, -0.40440401434898376, -0.662121593952179, -2.181912422180176, 0.0472152978181839, -0.6012957096099854, 0.5028010606765747, -0.7289372086524963, -0.49926483631134033, -1.0161786079406738, 0.7443501353263855, 0.7230129837989807, 1.2880936861038208, 1.6476202011108398], [2.2729945182800293, -0.006777357310056686, -0.5689036250114441, -2.041499137878418, -1.0118361711502075, -0.6340281963348389, 1.4412343502044678, -2.5068936347961426, 0.375969260931015, 0.7214213013648987, -0.08269581198692322, -0.3671095371246338, -0.31172096729278564, 1.0849924087524414, -0.39355573058128357, 0.5447525382041931, 0.5029905438423157, -1.5230306386947632, 1.3951094150543213, 0.5965358018875122, -0.02539958246052265, 0.40477317571640015, 1.2272354364395142, -1.1143825054168701, 0.07978679984807968, 1.1058971881866455, 0.5153540968894958, -0.24940161406993866, -0.6101164817810059, 0.4258357584476471, 0.2344687283039093, -0.6730928421020508, -0.9005348682403564, -0.5899239182472229, -1.5630204677581787, -0.43101730942726135, -0.1966298222541809, 0.5787699818611145, -0.2556804120540619, 0.8393411636352539, -0.5717569589614868, -2.8085415363311768, 0.48209047317504883, -0.7922437191009521, -0.2809033989906311, -1.4455265998840332, 1.4298069477081299, 1.954393744468689, 0.9512761235237122, -1.2506365776062012, -2.2817153930664062, -0.808798611164093, -0.6503036022186279, 0.31462711095809937, 0.8905078768730164, 1.6470675468444824, -1.2641373872756958, 0.6185299754142761, 0.6127364635467529, -0.003708516014739871, -1.70572829246521, -0.6811551451683044, 1.1491526365280151, -0.9639497995376587], [0.6138075590133667, -0.7804048657417297, -0.5515170693397522, 0.5952267646789551, 2.082597017288208, 1.614495873451233, -0.13497214019298553, 0.651212215423584, 0.7504191994667053, 0.4524950087070465, 0.03291953355073929, 0.7788712382316589, 1.0630205869674683, 0.5939504504203796, 0.33959442377090454, -0.1677936166524887, -0.6985142230987549, -0.8203595280647278, 0.3564254939556122, 1.5751147270202637, -1.187698483467102, -1.364661455154419, 0.796835720539093, 0.4550289511680603, -1.9260238409042358, -0.6704905033111572, 0.5028225779533386, -1.1879826784133911, 1.3077137470245361, 0.11855137348175049, -0.24952280521392822, 1.0651865005493164, -0.11892619729042053, 0.0195216815918684, -0.15793731808662415, -0.2844080924987793, -0.8988754153251648, 1.604205846786499, -1.361932396888733, 0.4798542857170105, -0.8209543824195862, 0.24756698310375214, -0.8958733677864075, 0.24139179289340973, 0.9636626243591309, 0.060161322355270386, -0.044173434376716614, 0.07252555340528488, 0.9092724919319153, -0.9919822216033936, 0.061316922307014465, 1.1916968822479248, 0.7349315881729126, 0.5645561814308167, -0.3507443070411682, -0.9651309251785278, 1.5714330673217773, -0.07539904862642288, 0.0018416730454191566, 0.7765771150588989, 0.1893942952156067, -0.7300037145614624, 0.9438152313232422, -0.5033249258995056]], [[-0.006436409428715706, -0.19210349023342133, -0.8998745083808899, 0.5802339315414429, -1.351345419883728, 0.9874894618988037, 0.2844283878803253, -0.6952989101409912, 0.47960996627807617, -1.4776638746261597, 3.0881550312042236, -0.09718068689107895, -0.3245948851108551, -0.766045093536377, 0.8254354000091553, -0.27443382143974304, 1.20597505569458, 0.7353302240371704, 0.1459672749042511, -1.2241711616516113, -0.7552110552787781, -0.8402437567710876, 1.5246535539627075, 0.14366571605205536, 0.690287709236145, 0.6545976996421814, -1.2885453701019287, -1.5189073085784912, -0.6077395081520081, -2.701706886291504, -0.07415740191936493, -0.23164938390254974, -0.3874525725841522, 1.147491693496704, 0.7093045711517334, -0.21435318887233734, -0.39519333839416504, 0.19055485725402832, -0.1735822558403015, 0.522389829158783, 1.508697509765625, -0.12400813400745392, 1.2801620960235596, 0.7442874312400818, -0.031353216618299484, -0.20425771176815033, -0.8669273257255554, 0.531011700630188, -0.5894597768783569, -0.04701739177107811, 1.523608684539795, 1.3168299198150635, -0.19977059960365295, -0.599025547504425, 1.2069859504699707, -0.2901035249233246, 0.5785166025161743, -1.8968044519424438, 1.9741309881210327, -1.3445223569869995, 0.7773278951644897, 1.4107508659362793, -0.18872417509555817, 1.0036765336990356], [0.949551522731781, 0.36149320006370544, -0.4757373631000519, 0.02216433547437191, -1.5332553386688232, 0.6725612282752991, 1.0991212129592896, 1.7869211435317993, 0.6910924911499023, 0.08349140733480453, 0.743205189704895, -0.5366328954696655, -1.6528481245040894, -0.2322310358285904, 0.07270140200853348, 0.29145461320877075, 1.2608191967010498, 0.8187901973724365, 1.0071907043457031, 1.1828370094299316, -1.9676803350448608, 1.4319236278533936, 1.835747480392456, 0.055624570697546005, -0.18363741040229797, 0.5396456122398376, -0.009837500751018524, -0.5472429394721985, 0.39753827452659607, -0.9621347188949585, 1.387154221534729, -0.743014931678772, -0.8185508251190186, -0.4043969213962555, 1.699175238609314, -1.2105787992477417, 0.34095317125320435, -0.7850145101547241, -1.3987778425216675, 1.5268300771713257, 0.21507145464420319, 1.2769237756729126, 1.1635388135910034, -0.5320321917533875, -0.48314228653907776, -0.0567755401134491, -0.15821124613285065, 0.4968114197254181, 1.5424776077270508, -0.7670826315879822, -0.514076828956604, 1.2646218538284302, 1.4891350269317627, -0.3359571695327759, 0.3553617298603058, -0.2614453434944153, 1.641715168952942, 0.6055487394332886, 0.23574253916740417, 0.9461084604263306, 0.8661386966705322, 0.014353002421557903, 0.7728404998779297, 0.49074217677116394], [0.3015778660774231, 0.07186924666166306, 0.9797871708869934, 1.5035611391067505, -1.2079060077667236, 0.28200653195381165, -1.273789644241333, -0.49447232484817505, -0.8022061586380005, -1.1422147750854492, 0.3845566213130951, -0.07880831509828568, -0.2797462046146393, -0.2117997109889984, -0.7794296145439148, -0.25885093212127686, -0.6857959032058716, -0.1840469241142273, -0.5255358815193176, 0.09622218459844589, 0.21946051716804504, -0.14532431960105896, 1.2611384391784668, 0.282662957906723, -0.2267674058675766, 0.5794154405593872, -2.506547689437866, -1.1932603120803833, -0.28661951422691345, -0.37268608808517456, -0.7509964108467102, -0.47422224283218384, 0.4342886209487915, -0.580693244934082, 0.5164502859115601, 0.7713235020637512, -0.056726183742284775, 0.8488770127296448, -0.6729886531829834, 1.0417062044143677, 0.9070141315460205, 0.2591980993747711, 0.3750735819339752, -1.817098617553711, -1.0609043836593628, -0.8446151614189148, -0.023117007687687874, 0.41116437315940857, -0.23650896549224854, 0.6399446725845337, 0.579979419708252, -2.745056390762329, 0.2663349509239197, -0.1778038591146469, -0.900150716304779, -0.3102705776691437, 0.5989768505096436, 0.6780032515525818, -0.1633123904466629, -0.372787743806839, 0.4830649793148041, -0.7721054553985596, 1.585675835609436, -0.47952598333358765], [-0.03826751932501793, -1.62811279296875, -0.019252339377999306, 0.26920974254608154, -1.615921139717102, -0.4785243272781372, 0.6743799448013306, 1.2658787965774536, 0.6977770328521729, 0.2395053505897522, 0.0255263838917017, 0.8157777786254883, 0.20511503517627716, -0.5004540085792542, -1.8715367317199707, -0.938057005405426, 0.1222919225692749, 0.5792638659477234, -0.44415587186813354, -0.09548776596784592, 0.09961912781000137, -0.2438475340604782, -1.0604742765426636, -0.37043577432632446, 0.5151888728141785, 0.8797526955604553, -0.5194955468177795, -2.295898675918579, -1.4077913761138916, -2.059330701828003, -1.6580246686935425, 0.3398677408695221, -0.535933792591095, 1.3181771039962769, -1.0468428134918213, 0.03360067680478096, 0.14650218188762665, 0.3161962628364563, -1.7830109596252441, -0.38035619258880615, 0.9328957200050354, -1.824619174003601, 2.4176032543182373, 0.05620897561311722, 0.32961705327033997, 1.295322299003601, 0.076776884496212, 0.9897280335426331, -2.3055200576782227, -0.2163306623697281, -1.2842813730239868, 0.8382542133331299, -0.18160472810268402, -0.6860418915748596, 0.7826744318008423, -2.131570339202881, 1.0977790355682373, -0.8405455350875854, -0.6743128895759583, -2.415963888168335, 1.0965465307235718, 1.165540099143982, -0.2364213615655899, -0.48562493920326233], [0.13329526782035828, 1.8002870082855225, -0.6350920796394348, -0.5395434498786926, -2.5219082832336426, -2.1231887340545654, -0.3124339282512665, -0.5729196071624756, 0.8533511757850647, -0.5683585405349731, 0.4552546739578247, 0.49115702509880066, 1.9040759801864624, -0.44925013184547424, 0.3635425269603729, 1.065608024597168, -1.5035876035690308, 0.42536547780036926, -1.2618591785430908, -0.5671479105949402, -1.3753377199172974, -0.35356128215789795, 0.22020097076892853, 1.568140983581543, -0.2120458036661148, 0.04965806007385254, -0.7376015186309814, -0.11092355847358704, 2.0685744285583496, -0.14693346619606018, -0.0642213523387909, -0.8882770538330078, 0.5668491721153259, 0.816791832447052, 1.15696120262146, -0.47981634736061096, -0.6745662689208984, 1.2690201997756958, -0.025837915018200874, -0.027157265692949295, 0.7223012447357178, 0.22398890554904938, -1.1308881044387817, 0.922382116317749, 0.21909092366695404, -2.0574467182159424, -0.7202380299568176, -0.4272400140762329, -1.6312682628631592, -0.6428537368774414, -0.03276880085468292, 0.3281231224536896, 0.059970077127218246, -1.4508767127990723, 2.6595821380615234, 0.6596378087997437, 0.5055230259895325, 0.5309200882911682, -0.521687924861908, -1.4076288938522339, -0.19864284992218018, 0.20106665790081024, 0.1485731601715088, 0.5604261159896851], [1.090460181236267, -1.6225873231887817, -1.185227394104004, -1.2123740911483765, -0.14287538826465607, -0.41884875297546387, 0.06832647323608398, -0.4793623685836792, 0.42367202043533325, -0.3771705627441406, 0.5358504056930542, -1.0751312971115112, -0.15462976694107056, -0.9408121109008789, 0.5517317652702332, 0.23616020381450653, -0.293455570936203, -0.2943045198917389, 0.019552353769540787, 0.2648095488548279, -1.424153208732605, -2.477937936782837, -0.05648721009492874, 0.7093430161476135, 0.26831674575805664, -0.8508980870246887, 1.1323968172073364, -1.204554557800293, -1.5385011434555054, -0.9420133829116821, 0.542188286781311, 0.9586615562438965, -0.4668351411819458, -1.0935144424438477, -0.5600430369377136, 0.5475242137908936, 0.3658609986305237, 0.43610307574272156, 0.9139801263809204, 1.4618197679519653, 0.08562278002500534, -0.0794532299041748, -0.9460894465446472, -0.47479650378227234, -1.928501009941101, 0.03534305468201637, -1.7782341241836548, -1.2334063053131104, 0.7086160778999329, 0.16141599416732788, -0.3502286374568939, 0.18193255364894867, 0.7829709053039551, -0.6715757250785828, -1.5602309703826904, 3.1302731037139893, 1.1999797821044922, 0.23170535266399384, -1.0873098373413086, 0.9609846472740173, -0.6542973518371582, -1.119627594947815, -0.08508910983800888, -0.38308146595954895], [-0.2305998057126999, 0.3165000379085541, 0.7937287092208862, -0.44971442222595215, -0.9482747912406921, 0.37491917610168457, 1.401442050933838, 0.6051344275474548, 0.5308927297592163, -2.424321413040161, 0.8395231366157532, 0.770418107509613, -1.8848494291305542, 0.20809419453144073, 2.1188535690307617, 0.8030587434768677, -1.226096510887146, 1.1239272356033325, -0.6133702993392944, -1.0142648220062256, 0.01637391746044159, -1.2323949337005615, -0.975182056427002, 2.2155046463012695, -2.0845606327056885, 1.2404117584228516, 1.4000617265701294, -0.37422120571136475, 0.7019509673118591, -0.05947740748524666, -0.1372123807668686, 0.010648838244378567, 1.5789762735366821, 1.415154218673706, -0.034286078065633774, -0.7015349864959717, -0.7448868155479431, -0.6894184350967407, 1.5360785722732544, 0.39733636379241943, 1.6213339567184448, 0.6437059640884399, 1.279622197151184, -1.6277321577072144, 1.1801104545593262, -1.4723716974258423, 0.8876132965087891, 1.2666562795639038, -1.1273006200790405, 0.701999306678772, -0.10820434987545013, 0.08032199740409851, 1.393279790878296, -0.30645951628685, -1.2733657360076904, 1.3296735286712646, 0.8834463953971863, 0.24200454354286194, -1.9533261060714722, -0.7905541658401489, 0.4576599895954132, -1.7353435754776, 1.6527961492538452, 0.5227649807929993], [1.8772408962249756, -0.3607428967952728, 1.1871836185455322, -1.0809824466705322, 0.849565327167511, -2.007338285446167, -1.6672430038452148, -0.29643580317497253, 0.9007117748260498, -0.06763532012701035, -2.078707218170166, 0.11658883839845657, 1.7876547574996948, 0.1872405856847763, -0.5319313406944275, 0.378167062997818, -1.013427972793579, 3.1484878063201904, -1.5429593324661255, 0.672370970249176, -1.6586189270019531, 0.1415349394083023, -0.5258827805519104, 0.8967996835708618, -0.3060387372970581, 1.439342737197876, -1.6670314073562622, -1.00566828250885, 0.27699869871139526, 1.6274478435516357, 0.29713740944862366, 0.42946889996528625, -1.4270823001861572, 0.06918676942586899, -2.097381591796875, -1.010650396347046, -0.20759953558444977, 0.3976084887981415, 0.5717584490776062, -0.2961166203022003, 0.634975790977478, -1.0026532411575317, -0.8835422992706299, -1.600589394569397, 0.6874868869781494, 2.4308395385742188, -0.09795279800891876, 0.3059031069278717, 0.13476791977882385, -0.2318512350320816, -0.24081070721149445, -0.002210980048403144, -0.5007626414299011, -0.1948285847902298, 0.3569238781929016, 1.3185306787490845, -0.18599966168403625, 0.1894201934337616, 0.6132463216781616, -1.999625563621521, -0.39137783646583557, -1.1682031154632568, -0.7573682069778442, -0.06513028591871262], [-0.681043803691864, -0.7394797801971436, 0.8580759763717651, 0.3595867455005646, -0.5463042259216309, 0.7535814642906189, 0.07399840652942657, 1.1669539213180542, -0.6791395545005798, 0.4252035319805145, -0.4332229495048523, -0.7045774459838867, -2.104712724685669, -0.4828242063522339, 0.4678173363208771, -0.7499746084213257, -0.11358998715877533, 1.134636402130127, -1.4090512990951538, 0.6484652757644653, -1.9863054752349854, 0.7081549167633057, 0.5417935252189636, 1.7500865459442139, -0.25813916325569153, 0.3130358159542084, -1.8116217851638794, 0.9230989217758179, -1.09224534034729, -2.4940474033355713, 0.14580382406711578, 0.7452004551887512, -1.2160990238189697, -0.2841338515281677, -0.14315176010131836, 0.8995679616928101, 0.5438368916511536, -0.6116459965705872, 0.6913975477218628, 0.6102914214134216, 1.380023717880249, 0.8317825198173523, 0.844399094581604, -0.5000507831573486, -0.3054315447807312, 1.4194566011428833, -0.7809796333312988, -0.03609707951545715, 1.148540735244751, -0.7827228903770447, 0.49925416707992554, -0.00792770180851221, 2.486891031265259, 2.083360433578491, 1.066318154335022, -0.7445684671401978, 1.128781795501709, -1.3605412244796753, 0.9026252627372742, -0.4592742919921875, -1.5610450506210327, 0.4416603446006775, 0.27717286348342896, 1.0924649238586426], [0.4453706443309784, 0.7495551705360413, -0.37009501457214355, -1.2766592502593994, -0.3654617369174957, -0.32780227065086365, -0.11832985281944275, 1.545218586921692, -0.43471601605415344, 0.1806495636701584, -0.21142084896564484, 0.6488543152809143, -1.2958629131317139, 0.12964583933353424, 1.0713061094284058, 0.5835492610931396, 0.8730122447013855, -0.11535753309726715, -0.5229129195213318, -1.2422469854354858, 1.198825716972351, -2.14821457862854, 0.33142945170402527, -0.7513898015022278, 1.1046892404556274, 0.11005441844463348, 0.11035091429948807, -1.1970651149749756, 1.748967170715332, 0.9211442470550537, 2.4682059288024902, 1.4823416471481323, -0.14949621260166168, 1.2571223974227905, 0.3519109785556793, 0.5845890045166016, -0.17625723779201508, -0.3021785616874695, 1.0585111379623413, 1.3177180290222168, -1.418432593345642, -0.01606173999607563, -1.1966780424118042, -0.022009264677762985, 0.2771218419075012, -1.034251093864441, 1.7580187320709229, 1.5240283012390137, 0.34167835116386414, -0.10496275871992111, 0.768083393573761, -0.036554962396621704, -0.5587934851646423, -0.49697786569595337, 1.4584616422653198, -0.07241977751255035, -0.45716455578804016, -0.5440634489059448, 0.8486869931221008, 0.3137184679508209, 0.9482271671295166, 0.6542477011680603, 0.9472557306289673, -0.002082705730572343], [-2.1762893199920654, 0.8430951237678528, -0.5278239846229553, -0.5037903189659119, -0.7067850828170776, 0.832605242729187, -0.5156419277191162, -0.25618335604667664, -2.155061721801758, -0.40895527601242065, 0.9542035460472107, -0.19625233113765717, 0.39355164766311646, 0.9318352937698364, 1.2681671380996704, -0.020871581509709358, 1.2236398458480835, 0.20851777493953705, -0.21582375466823578, 0.9442683458328247, -1.7584612369537354, 0.697783887386322, 0.7590191960334778, -0.9836622476577759, 1.1162962913513184, 0.09802883118391037, 0.1324395090341568, -1.6504253149032593, 0.21562252938747406, 0.19245220720767975, -0.641697347164154, -0.3091772794723511, 0.3870202898979187, -0.6215901374816895, 0.23426738381385803, 0.6473957300186157, 0.19798831641674042, -0.14414553344249725, -0.1719684600830078, -2.0268261432647705, -0.8903615474700928, 0.7742050290107727, -0.011490018106997013, -0.8855294585227966, 0.1899072378873825, 1.0684101581573486, 0.06763599067926407, -0.8227362632751465, -0.43021780252456665, -1.0448187589645386, 1.1578365564346313, 0.8564876914024353, -0.5444551706314087, 0.5349248647689819, -0.9545013308525085, 0.07158441096544266, -2.0629913806915283, -0.44432005286216736, 1.756797432899475, 1.2428268194198608, -0.1959114372730255, 0.4423297345638275, -0.8889158964157104, 0.3318384885787964], [0.03841913491487503, 0.2599940896034241, -0.08467300981283188, -1.4068386554718018, -0.909942090511322, -0.25046834349632263, 0.9637173414230347, -2.499584913253784, -1.0374373197555542, -0.919711709022522, -0.2388239949941635, 0.05966068431735039, -2.157559633255005, 0.882551372051239, 1.3213558197021484, 1.547763466835022, 1.4913614988327026, 1.8407312631607056, -0.48469069600105286, 0.23360490798950195, -0.2500046193599701, 0.021403564140200615, 0.8932451605796814, 0.11657141149044037, 1.594622254371643, 2.0259149074554443, -2.510436773300171, 1.0999785661697388, -0.09135712683200836, 2.0110254287719727, 1.083349347114563, -0.5102695822715759, -1.172066330909729, 0.658111572265625, 0.5630711317062378, 1.5032364130020142, 0.5958184599876404, -0.8393847346305847, 0.1517830640077591, 1.4063489437103271, -0.32552722096443176, -0.8105606436729431, 1.6118282079696655, -0.4306069612503052, 0.5698582530021667, 0.319499671459198, 2.391166925430298, 1.939727783203125, 2.1000423431396484, -1.089426875114441, 0.029925765469670296, -3.237097978591919, -1.2415090799331665, -1.035822868347168, 0.35897842049598694, -0.6894528865814209, -1.8447669744491577, 1.3842267990112305, 0.05331944301724434, -0.3838063180446625, -1.3463939428329468, 0.12829647958278656, -0.06354330480098724, 0.9389973878860474], [-1.3487437963485718, -0.20119349658489227, -0.5453379154205322, -1.3120907545089722, -0.06286722421646118, -0.28020310401916504, -1.661022424697876, 0.6705498695373535, -0.11133460700511932, -0.5132566690444946, -1.0536648035049438, 1.2347701787948608, -0.1316789984703064, 0.5732832551002502, 0.8519999980926514, -1.0469751358032227, 0.5348936319351196, -0.3932737708091736, -0.308383971452713, -1.8786742687225342, 0.13592813909053802, 0.23692595958709717, -0.7138814330101013, -1.2337130308151245, -1.2072854042053223, 0.1382429301738739, 0.4027647078037262, -0.48216959834098816, -1.1589651107788086, 0.41897305846214294, 0.3565202057361603, 0.7240332961082458, 0.23371078073978424, 0.44661104679107666, 0.4232490658760071, -2.3353428840637207, -1.4565515518188477, 1.4148139953613281, 0.3213365375995636, 1.5128859281539917, 0.7221654057502747, 0.1881287842988968, -0.2084309458732605, 0.5086467862129211, 1.2553120851516724, 0.03273915499448776, 0.4585888385772705, -0.4154229164123535, -0.2843814790248871, 0.10819333046674728, 0.5538269281387329, 0.8146688938140869, 0.5393084287643433, -0.7363662123680115, 0.0014896474312990904, -0.2859335243701935, 0.7677155137062073, 1.2323129177093506, 1.731413722038269, 0.3587796986103058, 0.4814854562282562, 0.11059239506721497, 1.0226233005523682, -0.6950884461402893], [0.5710785984992981, -0.28966471552848816, -0.9087542295455933, 1.354811429977417, -0.6004570722579956, -0.3725983500480652, -0.3823675215244293, 0.5808518528938293, 0.0542878732085228, 0.5156216025352478, -0.12940675020217896, 1.1257774829864502, -0.7015636563301086, 0.9896633625030518, -0.03017430752515793, 1.2266809940338135, -0.057797010987997055, -1.5641816854476929, -1.4538809061050415, -0.15181460976600647, 0.00856236182153225, -0.5070018768310547, -0.5758419036865234, 0.051470428705215454, -0.7030662894248962, -1.461174488067627, 0.4768659472465515, -0.6257140040397644, -0.2787543833255768, -2.3807389736175537, -0.3405193090438843, 0.1138676255941391, 0.7679755091667175, 0.3494984209537506, 0.09853767603635788, 1.107571005821228, -1.252393364906311, 1.3363733291625977, 0.6877326965332031, -0.8276675939559937, 0.7465289831161499, 0.7015155553817749, -0.25680118799209595, -1.7247508764266968, -1.4337880611419678, -0.5965878367424011, -0.04603644460439682, -0.26955074071884155, 1.3205194473266602, -0.9591459631919861, -0.07210661470890045, -1.4378461837768555, -0.5291209816932678, 0.21893540024757385, -0.5559883117675781, 1.250417709350586, 0.19884736835956573, 1.1518583297729492, 0.8827476501464844, 0.37402522563934326, 0.014726463705301285, 0.2600557506084442, -0.024775667116045952, -1.107936978340149], [-0.3437410295009613, 0.02297825925052166, 0.5949034094810486, 0.3685712218284607, 1.1808360815048218, -1.2164233922958374, 2.0662827491760254, -0.6945061087608337, 0.028360804542899132, -0.33831945061683655, -0.8323983550071716, -0.9560925364494324, -0.044615983963012695, 0.9452469348907471, -0.410899817943573, -1.5000317096710205, 1.620188593864441, 0.7094306945800781, -0.12925086915493011, 1.6813782453536987, 0.6914003491401672, 0.3524070680141449, 3.1934125423431396, -0.6605128645896912, 0.6267614364624023, 0.23094087839126587, -0.7421813607215881, -1.3822264671325684, -0.8329029679298401, 0.23070353269577026, 1.6225011348724365, -0.713153600692749, 0.8700149059295654, -0.2100970447063446, -0.18142203986644745, -0.6496198177337646, -1.5872172117233276, -1.0488898754119873, -0.9381593465805054, -2.842759847640991, 0.8368681073188782, 0.028677061200141907, -0.18406642973423004, 1.8338898420333862, -2.1780264377593994, 0.11727269738912582, 0.8581454753875732, 0.19377568364143372, 0.09343480318784714, -0.16093988716602325, -0.7481544017791748, -2.0153238773345947, -2.4516570568084717, 1.203567624092102, 0.772503137588501, -1.3143556118011475, 1.712141990661621, 0.19064664840698242, -0.09328147768974304, -0.05452299118041992, -1.1874756813049316, 0.4720016121864319, -1.3606352806091309, -0.6370797753334045], [-0.4455939531326294, -0.8042545318603516, -0.9968996047973633, -0.1335381418466568, 1.346902847290039, -0.522664487361908, 2.339191436767578, -1.453940749168396, 1.7359800338745117, 0.09634184837341309, -1.9976578950881958, 0.2555497884750366, -0.15037421882152557, 0.31652122735977173, -0.14571528136730194, -0.5591779947280884, -0.07957256585359573, 0.5704148411750793, -1.1486917734146118, -0.03263389691710472, 0.283066064119339, -1.1961253881454468, 1.7360948324203491, 0.5587741732597351, -0.22713139653205872, -2.375276803970337, -0.0750146433711052, 2.6668906211853027, 0.5919290781021118, -0.5628178715705872, -0.5953052639961243, -1.1277886629104614, 0.5829381942749023, 0.5790450572967529, -0.49859076738357544, -0.6439550518989563, -0.09932250529527664, -0.8882065415382385, 1.0022448301315308, 1.4307416677474976, -0.30198681354522705, 1.9124114513397217, 1.451066017150879, 0.3010737895965576, -0.40974950790405273, 1.393506646156311, -1.6219522953033447, 0.031126560643315315, -0.818152666091919, -0.5392107367515564, 1.1037770509719849, -1.3988429307937622, 0.3470095098018646, 1.547764539718628, -0.7479811906814575, -0.9787521958351135, 2.14506196975708, -1.9010043144226074, -0.03685692697763443, 0.22796276211738586, -1.0232881307601929, 0.2882900834083557, -1.078616976737976, -0.5597847700119019]], [[0.2355152666568756, -0.10466021299362183, 1.018897294998169, 1.1582082509994507, 1.2240198850631714, -2.305635929107666, -0.38370293378829956, -0.4294619858264923, 0.38523873686790466, -1.0892313718795776, -1.3606318235397339, 1.0985685586929321, -1.5069268941879272, -0.4109344184398651, -1.6072925329208374, 0.8558081984519958, -1.0988731384277344, 0.06516758352518082, -0.03437710925936699, 1.5303434133529663, 0.3795037269592285, -1.0078685283660889, -2.1212127208709717, 0.12244363129138947, 0.898246169090271, 2.5808053016662598, -0.5898218750953674, -1.0552802085876465, -0.9017239809036255, 1.8048564195632935, 0.3096015453338623, -0.5847292542457581, 1.2084745168685913, 0.666142463684082, -1.80678391456604, -2.346839427947998, 2.1404495239257812, 0.22265495359897614, -1.0631961822509766, -0.5561016201972961, 1.020684838294983, -0.8692542314529419, 1.166593074798584, 0.15492203831672668, -0.07452018558979034, -0.21600651741027832, -0.907005250453949, -2.2129738330841064, 1.2022264003753662, -0.4867984652519226, -0.49970942735671997, -0.8095400929450989, -0.9354225993156433, -0.6526533961296082, -2.286027431488037, -0.34734123945236206, -0.07636955380439758, -1.228002905845642, 0.2995317578315735, -0.053864043205976486, -0.8011204600334167, -0.7998402714729309, 0.4595440924167633, -0.5411890149116516], [-1.1852163076400757, 0.7220003008842468, -0.19557823240756989, -0.06289451569318771, -0.9077610373497009, -0.039553362876176834, 1.0096153020858765, 0.11880114674568176, 0.3389618992805481, 0.09369229525327682, 0.11920144408941269, -2.2166197299957275, 0.571575939655304, 0.9186254143714905, -0.01948355883359909, 0.7101342678070068, -0.2000073790550232, 1.0655289888381958, -1.45989990234375, 0.4449117183685303, 1.0168888568878174, 1.100903034210205, 0.8990696668624878, 0.6015706062316895, -0.569839596748352, -0.8413155674934387, 0.6515614986419678, -0.46474340558052063, 0.43897736072540283, -1.1306158304214478, -0.018131013959646225, -1.247489333152771, -1.2351044416427612, 0.19654186069965363, 1.4210022687911987, 0.15847769379615784, 1.080812931060791, 0.8133547902107239, 0.4611338973045349, -0.28154513239860535, -0.0057786270044744015, 1.0169602632522583, 1.3641228675842285, 0.261398583650589, 0.6063838005065918, 1.656334638595581, 0.37323710322380066, -0.6746522188186646, -0.6111969351768494, -1.1284406185150146, -1.5450615882873535, -0.3589199185371399, 0.08049242943525314, 0.2774556577205658, 0.5133093595504761, -0.0071625555865466595, 1.681113362312317, -0.8536428213119507, -0.06854093819856644, 1.817050814628601, 1.148462176322937, -0.26195311546325684, 2.700927495956421, 1.128654956817627], [-0.7636023163795471, -0.4000815749168396, 0.9187595844268799, -1.0967029333114624, 1.2058603763580322, 0.8721967339515686, -0.0006915118428878486, 2.3346402645111084, -1.1848078966140747, -0.8852417469024658, 1.0132436752319336, -0.8653731942176819, 1.3483718633651733, -0.7727973461151123, 1.400608777999878, 0.9723018407821655, 0.3057946562767029, -0.631202757358551, -0.7050416469573975, -0.02699524164199829, -2.0698177814483643, -0.4086543917655945, -0.37402039766311646, -0.18614168465137482, -0.9203819632530212, 1.0653752088546753, 0.8015848398208618, -0.4272143542766571, -0.3788924515247345, -1.8399760723114014, -0.36493492126464844, -0.15247312188148499, 1.1425548791885376, 1.3177462816238403, 0.7021995186805725, 0.04053781181573868, 0.9344161748886108, 1.1478631496429443, -0.6211599111557007, -0.7119845747947693, 0.5744881629943848, 0.2982548475265503, 0.0699075236916542, -0.07626360654830933, -1.1729828119277954, -0.00889402162283659, -0.5172264575958252, -0.31898272037506104, -0.5373129844665527, 1.6840155124664307, 0.9934906959533691, 1.4300771951675415, 0.027159690856933594, 1.69199800491333, -0.13598187267780304, -0.13120831549167633, 1.1722607612609863, 0.6599289178848267, 0.15668673813343048, 0.5663874745368958, 1.5526169538497925, 1.3087997436523438, -0.1378554254770279, 0.3017064332962036], [-0.7815080881118774, -0.6397339105606079, 1.2991243600845337, 2.302767515182495, 1.2152118682861328, -0.4426921308040619, 0.9759657979011536, -0.15993165969848633, 0.5114891529083252, 0.6934341788291931, -0.8605181574821472, 0.078328438103199, 1.1562490463256836, -1.0534783601760864, 0.09976594150066376, 1.6745444536209106, 1.1166481971740723, -1.0369526147842407, -0.6111561059951782, -0.49639585614204407, 1.0230598449707031, -1.7828755378723145, -0.044819727540016174, 0.8330972194671631, -0.5081535577774048, 0.4921925961971283, -0.2717403471469879, -1.664930820465088, -1.478983759880066, -0.3359341323375702, 0.2919256091117859, 0.31349557638168335, -0.12405520677566528, 0.06165898218750954, -0.14662940800189972, -1.4545472860336304, 0.6847374439239502, 0.2328488677740097, 0.8717255592346191, 0.9553869366645813, 0.42307719588279724, 1.2786762714385986, 0.5301470160484314, 0.9403446912765503, 1.5291268825531006, -1.4496841430664062, -0.3073219060897827, 0.032451044768095016, 0.11779672652482986, -1.487315058708191, -0.6999922394752502, -1.1227694749832153, 0.995864987373352, 1.1141051054000854, -1.1550238132476807, -0.711844801902771, -0.336789608001709, 0.15347424149513245, 1.8503177165985107, 0.3265121579170227, -1.166179895401001, 0.5598036050796509, 0.4208827018737793, 1.7190502882003784], [-0.35265442728996277, -2.0046067237854004, -0.35208696126937866, -1.6630604267120361, 0.4189176559448242, -1.7192819118499756, -0.35066813230514526, -0.6714525818824768, 1.0108232498168945, 0.2513614296913147, 0.6211129426956177, 0.9662559628486633, -0.35268843173980713, -0.6183351278305054, -0.02415807731449604, 0.8801090717315674, -0.4323929250240326, 0.8845801949501038, -1.1226271390914917, -0.438388466835022, -0.44004005193710327, -1.3304346799850464, 2.8226332664489746, 1.9629794359207153, -2.3611128330230713, 0.626083254814148, -0.4053053557872772, 0.029835712164640427, 1.721416711807251, -1.1703336238861084, -0.8166236281394958, 2.3906311988830566, 0.38040152192115784, -2.1043806076049805, -0.19528444111347198, 0.48794955015182495, 0.2167923003435135, -0.3057870864868164, 1.7521730661392212, 1.7266849279403687, -0.6705009937286377, 0.036823879927396774, -0.006877924781292677, -0.17251266539096832, 0.4004230797290802, -0.03710700199007988, 0.760823130607605, -0.7045655250549316, 0.16393235325813293, 0.7780940532684326, 1.2976521253585815, 0.200094074010849, -0.7134799361228943, 0.05976198986172676, -0.06215853616595268, -0.44169294834136963, 1.0592342615127563, -0.9932650923728943, 0.7908799052238464, 1.2576022148132324, -0.9270119667053223, -2.506868839263916, -1.2900110483169556, 0.3879116475582123], [0.2481730580329895, 0.8019136190414429, 1.4687942266464233, 0.088346928358078, -1.1230177879333496, 1.7948241233825684, -0.15216708183288574, -1.162682056427002, -0.7705379128456116, -3.203611135482788, 0.9346267580986023, -1.3929674625396729, -0.16429947316646576, 0.482909232378006, -1.1580575704574585, -0.0644911453127861, -0.6113926768302917, 0.4281253218650818, 0.4809589087963104, -1.6169774532318115, -1.6387690305709839, 0.6629692912101746, -0.9192995429039001, 0.8788498044013977, 0.5492196679115295, 2.322754383087158, 1.8896843194961548, 1.7840583324432373, 0.013028088957071304, -1.2449047565460205, -0.029295388609170914, 0.8015261888504028, 0.35637539625167847, 0.42130938172340393, -0.6314601302146912, 0.06043488532304764, -0.09862279146909714, -0.1953173279762268, -0.08243486285209656, -0.6456506848335266, -0.07984679192304611, 1.0800100564956665, 0.6984004378318787, 0.5221428275108337, 0.3917410373687744, -0.982560932636261, -0.018626529723405838, 2.104579210281372, -1.1618541479110718, -0.2993517816066742, 1.5047529935836792, 0.9281363487243652, 0.1253480166196823, -0.7288944125175476, -0.25646331906318665, -0.3356490135192871, 1.2996364831924438, -0.7970079779624939, -0.36493757367134094, 1.4390039443969727, -0.6637768745422363, 0.4661581218242645, -0.8585401773452759, 1.9183331727981567], [0.35224252939224243, -0.30565160512924194, -1.3882670402526855, -0.5887367129325867, -1.4267572164535522, -0.6499752402305603, -0.8426721096038818, -0.711051881313324, -0.3614986836910248, -0.7582155466079712, -1.0370843410491943, -0.6709810495376587, -0.1699874997138977, 2.209792137145996, 1.6213496923446655, 0.5729888677597046, 0.06194286048412323, -0.784142792224884, -0.7921195030212402, -0.38309434056282043, 1.39462149143219, 0.36161094903945923, -0.8294634222984314, -0.3021176755428314, 0.23475787043571472, -1.1309995651245117, 1.3844953775405884, 1.1292433738708496, 0.975727915763855, 0.25299009680747986, 0.33454641699790955, -0.3347227871417999, 1.4940191507339478, 1.4967041015625, -1.3033537864685059, 0.004435290582478046, 1.4064921140670776, -0.31878677010536194, 0.10556435585021973, -1.37967050075531, 1.6942685842514038, 0.21903890371322632, -1.2424516677856445, -0.08696030080318451, 0.41935545206069946, 1.6877307891845703, -0.3518943786621094, -1.5901230573654175, -0.21932180225849152, 0.5374293923377991, 0.6455644369125366, 0.2535060942173004, 0.571008563041687, -0.11645543575286865, 0.6507523655891418, 1.7083611488342285, -0.4350177049636841, 0.7876835465431213, 0.9726076126098633, -0.785308301448822, 0.5665773153305054, -0.4676898121833801, -1.350888729095459, -1.6777900457382202], [1.1610807180404663, 0.005318419076502323, 0.5205300450325012, 0.7413027286529541, -1.797043800354004, 0.732356071472168, 0.48374104499816895, 0.39309120178222656, 0.2866845726966858, 0.9604052901268005, 0.15526893734931946, 1.3740177154541016, 0.22098194062709808, -1.3614742755889893, 0.007365422789007425, -0.8711652159690857, 0.46859514713287354, -0.6501113772392273, 0.011310956440865993, -2.076374053955078, 0.11300896108150482, 0.0966121256351471, 0.5412834882736206, -1.0005401372909546, 1.3985087871551514, 0.4019744396209717, 0.8648744225502014, -0.11547542363405228, -0.12926040589809418, -0.20138828456401825, 0.7550027370452881, -0.5417501330375671, 0.9982569217681885, -0.3862191140651703, -0.3780684769153595, 0.4547070264816284, 1.2317981719970703, -0.9838994145393372, 0.7332921028137207, 0.866224467754364, 1.310057520866394, -0.5629734396934509, 2.266310214996338, -0.25422680377960205, -0.18752393126487732, 0.799490213394165, 0.866707444190979, -0.2426072657108307, -0.13022522628307343, -0.09291639924049377, 0.43752360343933105, -0.764640748500824, -1.6411914825439453, -0.07410664856433868, 0.497384250164032, 0.462547242641449, -1.1113871335983276, 0.5406581163406372, 0.028073810040950775, 1.172421932220459, -0.5623722076416016, -0.9092110991477966, 0.9500689506530762, 1.3549482822418213], [0.5050193071365356, -0.42899179458618164, -0.012818953022360802, 0.011504920199513435, -0.8950791954994202, 0.7143340706825256, -0.5646110773086548, 0.9212552905082703, -0.38817551732063293, -0.2031262218952179, -0.0954568013548851, 0.19314981997013092, -0.137056365609169, 0.7592709064483643, -0.4875294864177704, 0.43236619234085083, 0.2931496500968933, -0.7610003352165222, -0.027659201994538307, 0.5244830250740051, 0.6689106822013855, 1.4425767660140991, 0.16226765513420105, -1.1880340576171875, 1.1668545007705688, -0.11634636670351028, 0.517680823802948, -0.733701229095459, 0.5591639280319214, 2.9635815620422363, -0.0349343977868557, 0.30032408237457275, -0.7179433107376099, 0.005347475875169039, -0.1697390079498291, -0.17198893427848816, -1.3562151193618774, -2.0751237869262695, 0.5058648586273193, -0.5651516318321228, -0.1718706339597702, 1.0086256265640259, 1.4777899980545044, 0.4495755136013031, -0.3590601682662964, 1.1205854415893555, 1.108406901359558, 1.1962984800338745, -1.698279857635498, -0.33790090680122375, -0.7600352168083191, -0.0470944419503212, -0.23024195432662964, 0.14464271068572998, 0.14293377101421356, 0.13588353991508484, 0.41819581389427185, 0.23910556733608246, 1.2475374937057495, 0.38230326771736145, -0.06756841391324997, -2.2658605575561523, 0.8525826930999756, 0.2471274733543396], [-0.03075227700173855, -0.7283056378364563, -0.06388308852910995, -0.3296146094799042, -1.8181040287017822, -0.3582853376865387, -0.004560456611216068, 0.5669193267822266, 0.5201267004013062, -0.2672313153743744, -0.054499488323926926, -0.4350196123123169, 0.5751049518585205, 0.37696805596351624, 0.08618863672018051, 1.4837883710861206, -0.22044780850410461, -0.22564300894737244, -1.0395375490188599, -1.6837308406829834, 0.07842221856117249, 0.6487431526184082, 1.4573014974594116, 0.5262570977210999, 0.07623102515935898, 0.26573267579078674, 1.749672293663025, -1.8540595769882202, 0.539430558681488, 0.7239049077033997, -0.46691200137138367, 1.5533430576324463, 0.4688219130039215, 0.2527543306350708, 0.010485775768756866, -1.7801593542099, 0.2780853509902954, -0.5115019083023071, 2.648609161376953, -0.908113420009613, 1.216646671295166, 1.3794232606887817, -0.5539999604225159, 0.19519011676311493, 0.6792540550231934, 0.2985866069793701, 0.7328551411628723, -0.9105565547943115, 1.622247576713562, -1.5166912078857422, 0.008435684256255627, -0.9346024990081787, 0.8483893871307373, 0.10429757833480835, -1.3613625764846802, 0.09186800569295883, -0.06933252513408661, 0.08150701224803925, -1.652418851852417, -0.7512048482894897, -0.02225092239677906, -0.5508002042770386, 0.7781292796134949, -2.0173637866973877], [-1.2362630367279053, -0.7839424014091492, 0.5346019864082336, 1.1146880388259888, -1.1201361417770386, 1.0688623189926147, -2.2410802841186523, -0.44269052147865295, -0.11167246103286743, 1.1365247964859009, 0.30417266488075256, -0.49903473258018494, -1.1720595359802246, 2.3471062183380127, -0.0023692382965236902, -1.4820362329483032, -0.3596392869949341, -0.3941253125667572, -1.3192769289016724, 1.6290634870529175, -0.38741034269332886, -0.5019455552101135, 1.5573312044143677, 0.9429478645324707, -1.466333031654358, -0.25475722551345825, 1.661340594291687, -0.8475913405418396, -0.7126779556274414, -1.7312828302383423, -0.8935274481773376, -0.008323295041918755, -0.47029051184654236, -0.32477623224258423, 0.31105127930641174, -0.07310274988412857, 0.6721614003181458, -0.3593936562538147, -1.741929054260254, 0.5425547361373901, 0.33885496854782104, -0.7054768204689026, -0.5643675923347473, 1.357082486152649, 0.6447165608406067, 0.4230227768421173, -0.6484017372131348, -0.886019766330719, -0.08373931795358658, -0.20803861320018768, 0.8945404887199402, -0.2093065232038498, 0.711242139339447, 1.3761060237884521, -1.9232115745544434, -0.225655660033226, 0.35879844427108765, 0.900184690952301, 0.9589802622795105, 0.318418949842453, 1.2086454629898071, -1.0038362741470337, -0.49217748641967773, 0.4536602795124054], [-0.6450989246368408, -0.40607768297195435, 1.7140796184539795, 0.19571521878242493, -0.021466312929987907, -2.1355245113372803, 1.5675420761108398, 0.5016597509384155, 0.17885151505470276, 0.3287722170352936, 0.40979334712028503, 0.3689192831516266, -0.7346307635307312, -1.475390076637268, 0.059034135192632675, 0.6832982301712036, -0.4764159321784973, -1.0408638715744019, 0.6437610387802124, -0.47490057349205017, -0.5786759257316589, 0.22058378159999847, 0.44708624482154846, -0.45717573165893555, -0.15838390588760376, -0.5290434956550598, -0.3796811103820801, -1.3916388750076294, -0.5394350290298462, 0.13712136447429657, -1.2523773908615112, 0.34221532940864563, -2.4640588760375977, 0.41912585496902466, 0.02429136075079441, -0.2040688842535019, -0.11484963446855545, -0.12170133739709854, -0.15936098992824554, 0.47947996854782104, -0.04216678813099861, 0.5619974136352539, 0.4339981973171234, -0.21225818991661072, 0.33997803926467896, 1.4102373123168945, 1.3056108951568604, 0.5827361941337585, -0.27992168068885803, -0.3503814935684204, -0.7396354675292969, 0.21085642278194427, -0.1729731261730194, -0.5056062340736389, -2.0189225673675537, 0.6051895022392273, -0.7771638631820679, 0.6152321696281433, 0.5374760627746582, 0.06952500343322754, 0.5101101994514465, 0.013681571930646896, 0.2614356279373169, 0.9345070123672485], [1.0036311149597168, -1.1893110275268555, -0.6094391942024231, -0.7335831522941589, 1.8046587705612183, 0.4233887493610382, -0.435857892036438, 0.40918293595314026, -1.763843297958374, 0.24183601140975952, 1.2419878244400024, -0.5490918159484863, 2.2509024143218994, 0.21209418773651123, -1.998389720916748, 1.24061119556427, 0.5311489701271057, 0.14085620641708374, 0.9320411086082458, -1.6973508596420288, -0.9744842052459717, -1.1783162355422974, -1.0987657308578491, 0.17970231175422668, -1.1525405645370483, -0.11741054058074951, 1.459389567375183, -1.2994853258132935, 1.2811726331710815, -0.03780724108219147, -0.7378501296043396, -1.7470027208328247, 1.1084916591644287, -0.1871887743473053, 0.5953169465065002, 2.185856580734253, -0.04848825931549072, -0.17810751497745514, 0.510911226272583, 0.7639049291610718, 0.23784103989601135, 0.6513977646827698, 0.4446111023426056, -0.15202315151691437, 0.2125740647315979, -2.061591863632202, -0.11736010015010834, 1.7085201740264893, -1.231358528137207, 0.31969428062438965, 2.2279865741729736, -0.403168261051178, 1.248816967010498, -0.24281658232212067, 0.08176281303167343, 0.9446675181388855, 1.3564056158065796, 0.07493522018194199, 0.1527075171470642, 0.017587868496775627, -0.10199666023254395, 0.8635730147361755, 1.2645801305770874, -0.4500119686126709], [0.8938802480697632, -0.21351832151412964, -1.0124115943908691, 0.9032676815986633, 1.1691336631774902, 0.2776249945163727, -2.4873268604278564, -1.1759724617004395, -0.18642841279506683, -0.13147835433483124, -0.5494755506515503, 1.8411405086517334, 0.5464442372322083, -0.5269930362701416, -0.1340171992778778, 0.715042233467102, -0.22194306552410126, 0.4480884373188019, 0.06042398139834404, -0.380185604095459, 0.5258457064628601, -0.7782179713249207, 0.3793010711669922, 1.6088671684265137, -0.9149744510650635, -1.7952697277069092, -1.2138651609420776, 0.8342794179916382, 1.2457205057144165, -0.39201197028160095, -1.5240355730056763, 0.9329840540885925, 0.7581905126571655, 1.164838433265686, 1.3388510942459106, 0.2912100553512573, 0.09656645357608795, 0.7458353638648987, -0.3530625104904175, -0.09672144055366516, -1.4016377925872803, -1.2599478960037231, -0.9648175835609436, 0.33155563473701477, 0.09652943909168243, 0.5067187547683716, -0.7637730836868286, 1.016149878501892, -0.7018784880638123, 0.4624287486076355, 1.499887228012085, -1.7926414012908936, -0.2934715151786804, 1.0812489986419678, -1.2361418008804321, -0.26186197996139526, 0.21480201184749603, -0.41425591707229614, -1.102096676826477, -1.3452072143554688, -0.18648676574230194, -0.06311892718076706, 0.4425467550754547, -0.658406674861908], [0.08517380803823471, -0.6672956943511963, -0.9488962888717651, -0.918231189250946, 1.0010746717453003, 0.6985029578208923, -0.5164437890052795, -0.9004988074302673, 1.7931090593338013, 0.8097476363182068, 0.6392858028411865, -1.374197244644165, 0.48916101455688477, -1.779259204864502, -0.00495326379314065, -0.7840290069580078, -0.5662937760353088, -2.0837912559509277, -2.616384744644165, 1.9870479106903076, -0.8051345944404602, -1.7109930515289307, -0.9351174235343933, 0.24303178489208221, 0.3660532236099243, 3.0032811164855957, -1.5214250087738037, -1.317401647567749, -1.0468637943267822, 0.10396909713745117, -0.8832909464836121, -1.2989553213119507, 0.1023198738694191, 0.11284578591585159, 1.4278358221054077, -0.7065039873123169, -0.8503172397613525, -0.021669358015060425, 0.19959446787834167, -1.0285242795944214, 0.516559362411499, -0.8485429286956787, -0.6614418625831604, -0.2807236611843109, -1.6923936605453491, -0.30464404821395874, -1.2771074771881104, -0.4098384976387024, -0.3285049796104431, -0.46341875195503235, 0.48273611068725586, 0.6540045738220215, 0.43536561727523804, -1.5498424768447876, 0.6216974854469299, -0.3803464472293854, 0.2519225776195526, -0.633558452129364, 1.7574831247329712, 1.0682469606399536, -0.982076108455658, -0.18745861947536469, 0.8883086442947388, 0.8633409142494202], [0.6609224677085876, 0.8467981219291687, -0.3080574572086334, 0.5401238203048706, 0.15196342766284943, -0.7358066439628601, -0.9339752793312073, 0.6294453740119934, 1.1387075185775757, -1.4808529615402222, 1.0722105503082275, 0.5030590891838074, -1.1106046438217163, -0.5401252508163452, 0.214365154504776, -1.1244714260101318, 1.0651721954345703, -1.245999813079834, -1.3482511043548584, 0.48705166578292847, 0.975320041179657, 0.01705063506960869, 1.082202672958374, 1.081252098083496, -0.8801530003547668, -0.18475545942783356, -1.288718581199646, -0.37309929728507996, 0.09388475865125656, -0.4156383275985718, 0.4410379230976105, 1.6862342357635498, 0.5609003901481628, 0.9252389073371887, 1.5535553693771362, 0.2255823165178299, 0.0924518033862114, 0.7105473875999451, -0.6388722062110901, 0.07624347507953644, 1.918531894683838, 0.4418470859527588, -0.10132946819067001, -0.2155132293701172, -0.7164098620414734, 0.638592541217804, -0.7072252035140991, -0.1969221532344818, 0.8838042616844177, -1.3381274938583374, -0.3568665385246277, 0.8409976959228516, -0.7613650560379028, -1.6123957633972168, 0.8349620699882507, -0.06892051547765732, -0.3536379337310791, -1.1833103895187378, 0.32228749990463257, -0.46550044417381287, 2.014253616333008, -0.29783958196640015, 1.0947548151016235, 1.834128975868225]]], [[[-0.4762415885925293, -1.1294341087341309, -0.21247495710849762, -0.42898041009902954, -0.8476241827011108, 0.2690390646457672, 0.49128231406211853, -1.7129861116409302, -0.7032782435417175, 1.1874048709869385, -0.13850142061710358, -0.8435003757476807, 0.5370597243309021, 0.3311793804168701, -1.5634294748306274, 0.40668338537216187, 0.4036882817745209, -1.4357783794403076, 0.5905762314796448, -0.23503117263317108, 0.3000231087207794, -0.10193075239658356, 0.19224029779434204, -1.3398174047470093, 0.6052118539810181, -1.8104068040847778, 0.4195577800273895, 0.6417385339736938, 0.7185598015785217, 1.859444260597229, 0.4561360776424408, -1.6145989894866943, -2.3203251361846924, 0.7940852046012878, 0.8595538139343262, -1.0104339122772217, 0.11108958721160889, -0.8131095767021179, -0.9745880961418152, -1.0890123844146729, 0.1905294507741928, 0.7711371779441833, 0.03747270256280899, -1.0958373546600342, 1.5862997770309448, -0.9303790330886841, -1.9091315269470215, 0.6627694368362427, -0.4248737096786499, 0.628926157951355, 0.6409845948219299, -0.16213923692703247, 0.8124426603317261, 0.45415735244750977, 0.22667072713375092, -0.5915957093238831, -0.647099494934082, 0.6921771764755249, 0.5393890142440796, -1.1179965734481812, 0.8633712530136108, -0.7922371625900269, -0.22585278749465942, 0.7980647683143616], [-0.3862917423248291, 0.7439643740653992, -0.16072995960712433, 0.6099331974983215, 0.0798906534910202, 0.08328339457511902, -0.7825604677200317, -0.761468231678009, 0.49353960156440735, -1.7980386018753052, 0.8130020499229431, 0.20971760153770447, -1.1610219478607178, 2.2991445064544678, 0.7003118991851807, -0.7059206366539001, -0.5815931558609009, 2.0905230045318604, -0.3264305889606476, 1.581035852432251, -0.3330623209476471, -0.41465049982070923, 0.8907473683357239, 1.471864104270935, 1.4099483489990234, -0.34973177313804626, 0.15611112117767334, -0.46167412400245667, -0.7871654629707336, 0.6582253575325012, 0.04616600275039673, 0.4215374290943146, -0.09934835135936737, -1.7085247039794922, -2.2827117443084717, -0.45034098625183105, -0.6211868524551392, 1.7170368432998657, 0.013188255950808525, -0.2699723243713379, 2.5362586975097656, 1.4395583868026733, 1.2122834920883179, -0.004815963562577963, -0.2411622703075409, 0.9320635199546814, -0.1740390807390213, 0.7321662306785583, 0.26024025678634644, -1.0914804935455322, 1.0510380268096924, 1.019715666770935, -1.8812322616577148, -1.0149978399276733, -0.3046368658542633, -1.446871042251587, 0.31546878814697266, 1.3865612745285034, 0.4332870543003082, -0.8467918634414673, 1.4929189682006836, 0.40209710597991943, 1.0653223991394043, -0.26222649216651917], [0.143632709980011, 1.5306618213653564, -0.016098273918032646, 1.428079605102539, -1.4361473321914673, 1.7034357786178589, 0.4895586371421814, -0.1972586065530777, 0.008450893685221672, -1.3310548067092896, 0.1663265824317932, 0.6423986554145813, 0.05844651535153389, 0.6274150013923645, 1.6686984300613403, -0.8892531394958496, 0.9662182927131653, 0.7164018750190735, -0.15893635153770447, 0.27579665184020996, 0.21471264958381653, 0.42408785223960876, -2.3359129428863525, -1.7102489471435547, 0.7719685435295105, -0.20924833416938782, 1.5480539798736572, 0.4066024720668793, 0.23276931047439575, -0.5396002531051636, 0.34889236092567444, -0.5868417620658875, 0.32107436656951904, 1.0014832019805908, -0.11186414211988449, -0.8859967589378357, 1.4647654294967651, 0.4268152713775635, -0.015542721375823021, 0.10455360263586044, 0.4014490246772766, -0.03947460278868675, -0.33685579895973206, 0.6683758497238159, -1.236083745956421, 0.40314093232154846, -0.4179278314113617, -0.70651775598526, -2.425980567932129, -0.02831961400806904, 0.8550817370414734, 0.8314409255981445, 0.15834493935108185, 0.08667359501123428, -1.3279775381088257, -0.27391576766967773, -0.41879427433013916, -0.31057482957839966, 0.6046705842018127, 0.5571097135543823, 0.9412106275558472, -1.9155558347702026, 0.33501243591308594, -0.12494775652885437], [0.30205732583999634, 1.559950590133667, 1.2861182689666748, 1.073791742324829, 0.6485075950622559, -0.14932069182395935, 1.0979597568511963, 0.38591647148132324, 0.1543866991996765, 0.4040150046348572, 0.5579670667648315, 0.7880021333694458, -0.43859124183654785, 0.34455612301826477, -0.10012178122997284, -1.6611989736557007, -1.1415984630584717, -0.30855080485343933, 0.4611462354660034, -0.7240217924118042, -0.4642195701599121, 0.2372167855501175, -1.0328267812728882, -1.0501829385757446, -1.407860279083252, 0.48047494888305664, 0.828476071357727, -0.13500449061393738, -0.9874749779701233, -1.370201826095581, -0.5215896368026733, 0.26811128854751587, 1.720645546913147, 0.0847875326871872, -1.1363400220870972, 0.5448320508003235, 1.6066285371780396, -0.3575778901576996, -1.4249018430709839, -0.8205471038818359, -0.5304379463195801, -0.31792759895324707, 2.5078554153442383, 0.4547836184501648, -0.2816382944583893, -0.47036072611808777, 0.8247971534729004, -0.888526201248169, -0.7356468439102173, 0.6751782894134521, 1.3900384902954102, -1.3126871585845947, -0.3445444107055664, 0.2549538314342499, 0.0863061398267746, 0.11701846122741699, -0.26954635977745056, -0.4984169900417328, 1.6758289337158203, -0.008772416971623898, -2.747875213623047, -1.8449991941452026, -0.0802355408668518, 0.29390421509742737], [-1.723604679107666, 0.32435110211372375, 0.5317571759223938, 0.2557806074619293, -1.6291117668151855, 1.0580755472183228, 0.21237513422966003, 0.3455091416835785, -0.5481398701667786, 0.87593674659729, 0.958031415939331, -0.9183218479156494, -1.063617467880249, -0.3847521245479584, 0.24255257844924927, 2.3083078861236572, 1.6457295417785645, -0.6962448954582214, -0.8207870125770569, 1.1933016777038574, -0.8854057192802429, -0.5937576293945312, -1.0168501138687134, -2.320180892944336, -0.6084939241409302, -1.0584784746170044, 0.5258122086524963, -0.6521655321121216, 0.3272555470466614, -1.3291211128234863, -0.4645509123802185, 1.7752166986465454, 1.8222780227661133, 1.0957294702529907, 0.7168442606925964, 1.1412544250488281, -0.24856425821781158, 1.9361367225646973, -1.1124134063720703, 0.7853408455848694, -0.23620261251926422, -0.693448543548584, 1.7908108234405518, -0.19524356722831726, 0.003998965956270695, -1.3709681034088135, -0.945660412311554, 0.5859221816062927, 1.0432722568511963, 1.3299570083618164, -0.0692039281129837, -0.13148801028728485, -0.8448122143745422, -0.3153878450393677, 1.1202484369277954, 0.7685649394989014, 0.6582436561584473, 0.39151668548583984, -0.911592423915863, 1.3432942628860474, -2.370666742324829, 0.6218851208686829, -0.604127824306488, 0.8472200036048889], [-0.28677815198898315, -1.0417219400405884, -0.1925656944513321, 1.218941569328308, 0.41332077980041504, 1.8973381519317627, -1.2058589458465576, -0.13956525921821594, 0.3007836937904358, -1.0207626819610596, 0.33607175946235657, -0.35614386200904846, 1.1699098348617554, 1.3759185075759888, -0.4191552400588989, -0.1243908479809761, 0.35463830828666687, -0.0124129056930542, -0.4547426700592041, -0.01268650684505701, -0.18948213756084442, 1.1406418085098267, 0.24661171436309814, -0.04401659220457077, 2.0610852241516113, 0.2509840428829193, -0.9167769551277161, -0.22323380410671234, 0.22592468559741974, 0.6270700693130493, 1.1356302499771118, -1.1813513040542603, -1.0789328813552856, -1.28115975856781, -0.12573328614234924, -1.5918993949890137, -0.8885741233825684, -0.09836045652627945, -0.3136981129646301, -0.5207041501998901, 0.5640081167221069, -0.15168772637844086, 1.0001767873764038, 0.3849373459815979, 0.32138559222221375, -0.1464228630065918, -0.48383358120918274, -0.055447883903980255, -0.3583614230155945, -1.9785770177841187, 1.360102891921997, -1.778670310974121, 0.6426130533218384, -1.2059820890426636, -1.5556583404541016, 0.35200774669647217, 1.3340176343917847, -0.729513943195343, 0.857189953327179, 1.2438385486602783, -1.5137661695480347, -0.32849475741386414, -0.5322401523590088, 1.1194757223129272], [0.9658584594726562, -0.6751036047935486, -0.5098749995231628, 0.9604613780975342, -0.7421318888664246, -0.12002070248126984, 0.48861128091812134, -0.0715683251619339, 0.8784609436988831, 1.2389569282531738, -0.42734524607658386, -0.7081524729728699, 0.33672651648521423, -0.3365502953529358, -0.27660053968429565, 0.8293609023094177, -0.6356480121612549, -0.632666289806366, -0.1256788969039917, -0.33052605390548706, 0.9723623394966125, -0.8063360452651978, 0.26080843806266785, 0.22592607140541077, -0.7011193037033081, 0.949497401714325, -1.6515504121780396, 1.3007862567901611, -0.3633144199848175, 1.0082752704620361, 0.16545046865940094, -2.294628381729126, -1.548762559890747, 0.07718274742364883, 0.6264434456825256, 0.9618324041366577, 0.6013339757919312, 1.0902822017669678, 0.319713830947876, 0.6904246807098389, 0.8293899893760681, 0.989339292049408, -1.7681350708007812, -0.6711991429328918, 0.03544631972908974, -2.023785352706909, 1.6273491382598877, 1.853161334991455, -1.325148105621338, 0.27561673521995544, 0.3855971693992615, 1.678254246711731, -0.95377516746521, -0.17399999499320984, -0.2985082268714905, 0.5111009478569031, -0.6698442101478577, -0.5877992510795593, -0.9608643054962158, -0.46770498156547546, -1.2834140062332153, -1.3173820972442627, -0.21646827459335327, 0.4247101843357086], [1.8440335988998413, 0.31369614601135254, 0.004935725126415491, -0.93315190076828, -0.6958403587341309, 1.137520670890808, 0.35382628440856934, 0.04385925829410553, 1.7597851753234863, 0.4795789122581482, 0.2006072849035263, -1.5750908851623535, -0.06546543538570404, 0.9234901070594788, -0.07217028737068176, -2.9055230617523193, 0.24730825424194336, -1.2901413440704346, -1.8710249662399292, 1.038670301437378, -0.7922561168670654, 0.2113012969493866, -2.575920820236206, -0.25351908802986145, 1.3919432163238525, -0.8807618021965027, -0.08736657351255417, 0.06848340481519699, 0.019268836826086044, -0.5230552554130554, -1.7033607959747314, -0.7273638248443604, 2.7893178462982178, 0.48555848002433777, 0.6625788807868958, -0.6081266403198242, 1.316060185432434, 0.1954440176486969, -1.2088415622711182, -1.5025930404663086, -0.07431351393461227, 0.5113115906715393, -0.0031972010619938374, -2.0577023029327393, 0.08627954125404358, 0.069468192756176, -0.8043855428695679, -1.7568104267120361, 0.037218254059553146, -0.823927104473114, -0.3452894687652588, -0.44319602847099304, 0.007813596166670322, -1.3258168697357178, -1.2225161790847778, 3.0751595497131348, 1.1606489419937134, -0.4211746156215668, 0.421008437871933, 0.6251091957092285, 0.057643450796604156, 0.16526901721954346, -0.005918701179325581, -1.1758065223693848], [0.10166596621274948, -0.03705182299017906, -0.5845822691917419, -3.0685813426971436, -0.6174756288528442, -0.850609302520752, -1.2262108325958252, -0.48311781883239746, -0.5670138001441956, 0.7630916237831116, -1.2396982908248901, -1.6077989339828491, 1.4142805337905884, -0.5687341094017029, -0.27396735548973083, -0.8039740324020386, 1.517831802368164, -0.552257239818573, 1.4669610261917114, 0.6850219964981079, -1.0561978816986084, -0.5002845525741577, -0.7253217101097107, 0.8995784521102905, 0.7391057014465332, -1.0004256963729858, 1.4866036176681519, -0.39933642745018005, -0.05713127180933952, 0.8989756107330322, -0.7438763380050659, -0.3345828652381897, 1.3975491523742676, 0.227035790681839, 1.3320515155792236, -0.16710717976093292, 0.02397344820201397, 1.1490973234176636, 0.8759214878082275, 0.6438634395599365, 1.0797854661941528, 1.1847952604293823, -0.6626052260398865, 1.7174837589263916, 1.2196929454803467, -0.37052974104881287, 1.8966296911239624, -0.03482028841972351, 0.145647794008255, 0.2433149665594101, -0.29262736439704895, 1.0897963047027588, 0.15232127904891968, -0.46935200691223145, -0.6761954426765442, -0.2671128511428833, -0.2204882949590683, -0.37149637937545776, -0.10110887885093689, 0.047425027936697006, 0.05803623050451279, 1.5216599702835083, -1.2568222284317017, 0.21738825738430023], [-1.357856273651123, 0.4645688533782959, 1.153469204902649, -0.8700218200683594, 0.8766265511512756, -0.4242557883262634, 1.1897310018539429, -0.11918331682682037, -0.5133398771286011, 0.26278355717658997, -0.5568469762802124, 0.1768079698085785, -0.42632657289505005, -1.12477445602417, 0.16196423768997192, -0.6732879877090454, 0.5422975420951843, -2.2567358016967773, 0.49037376046180725, 1.5778157711029053, 0.8995856642723083, -0.5922080278396606, 2.260495662689209, -0.7389471530914307, -0.4260227680206299, -1.4864888191223145, 0.3556804358959198, 1.2065757513046265, 0.2582765519618988, -0.3066854774951935, 0.45012718439102173, 1.8306885957717896, -0.33616557717323303, -0.6081551313400269, 0.8971949219703674, 0.6376999616622925, 0.4178880751132965, 1.1323450803756714, 1.2292892932891846, 0.19850090146064758, 1.4636200666427612, 0.2851715087890625, 0.5078747868537903, -1.2333729267120361, 0.7668204307556152, 1.6054571866989136, 2.0069189071655273, 0.1948077380657196, 1.2074865102767944, 0.8287904858589172, -0.8990893363952637, 1.1685154438018799, 1.667115569114685, -0.3729262351989746, -2.3348662853240967, 0.060719314962625504, 0.10121111571788788, 0.3301346004009247, -0.1580895483493805, 0.09550297260284424, -0.3777310252189636, 1.6864619255065918, 0.26679953932762146, 0.0056963274255394936], [1.159936785697937, 0.5093096494674683, 1.094188928604126, 0.001582933240570128, -1.244504690170288, 1.3681164979934692, -0.5618487596511841, -0.19223305583000183, -0.5430223941802979, 0.0007286079344339669, -0.5604934692382812, -0.4527803361415863, -0.8770367503166199, -0.1989872306585312, 0.10009349137544632, -0.3217376172542572, -0.8913240432739258, -0.3871554136276245, 0.44998791813850403, -0.41296350955963135, 0.6021215915679932, -0.7129284739494324, 0.8341543674468994, 0.6412782073020935, -1.363153100013733, 1.0759997367858887, -0.004521167371422052, 1.536157250404358, 0.4214830696582794, 2.218536138534546, -0.12124619632959366, 1.7567181587219238, 0.5056160688400269, 1.914907455444336, -2.127030849456787, -1.5661568641662598, 0.32631054520606995, -0.977749228477478, 0.04054897278547287, -1.9019020795822144, -0.3933776319026947, -1.1248736381530762, -1.5151065587997437, 0.08564125746488571, 1.060925006866455, -0.44387465715408325, -0.6630865335464478, 1.4887621402740479, 0.7375043034553528, -0.43229836225509644, 0.7972471714019775, 0.6435408592224121, -0.9106280207633972, 0.7571913599967957, -1.6907764673233032, -0.4833317995071411, 1.076317310333252, -0.11018919944763184, -1.8363146781921387, 0.4409528374671936, -0.3000720739364624, 0.058090660721063614, 0.35573282837867737, 0.04187316447496414], [0.14524568617343903, -0.642185389995575, 0.8101137280464172, 0.9152015447616577, 0.522257924079895, -1.7012395858764648, -0.7235016822814941, 0.19435696303844452, -0.129092738032341, 0.07468587905168533, -0.5676997900009155, -0.800080418586731, -0.8425219058990479, 0.6163857579231262, -0.511121392250061, 1.2098175287246704, 0.11697308719158173, -0.1695384681224823, -1.5445420742034912, -0.043668828904628754, 2.932023763656616, -1.6199544668197632, -1.4393125772476196, 0.1328403353691101, 0.23108476400375366, 2.0288784503936768, -1.1050161123275757, 0.9563701152801514, -0.042367007583379745, -1.4035325050354004, -0.7240862846374512, -0.6174461841583252, -1.3713726997375488, -0.6203316450119019, 0.17085573077201843, 0.9227756261825562, 1.9984066486358643, -1.1508252620697021, -0.026624029502272606, 0.00667658681049943, 1.4668930768966675, -1.009785532951355, 0.027655145153403282, -0.4879920780658722, 0.04027874022722244, 0.029071036726236343, 0.04444678872823715, 1.191707968711853, 0.9147062301635742, 0.303943008184433, 2.8593411445617676, 2.1134707927703857, 2.088339328765869, -0.3838246464729309, -0.24981889128684998, 2.458608865737915, -1.1768428087234497, 0.310346782207489, -0.11239097267389297, 1.8413317203521729, 0.6471856832504272, -0.14895811676979065, 0.5357813835144043, -0.7011310458183289], [0.6374489068984985, 0.032515749335289, 0.2301243394613266, -0.4914436638355255, 1.7660959959030151, -0.9013055562973022, -0.773421049118042, 0.3440929651260376, -0.14628159999847412, 0.3290520906448364, -0.12166589498519897, 1.4543037414550781, -0.12623412907123566, -0.10673730075359344, -0.6792982220649719, 0.6419416666030884, 0.1418379247188568, 0.6416435837745667, 2.048102855682373, 0.01494115125387907, 1.3492604494094849, -0.4058651328086853, -0.7787484526634216, 0.6092599630355835, 1.5161385536193848, -0.2024347186088562, -0.22748486697673798, -2.487401008605957, 1.6103267669677734, -1.1822941303253174, -1.2483432292938232, 0.1648050993680954, 0.7052446603775024, 0.6913759708404541, 1.1654400825500488, 0.9405807852745056, -0.10978706181049347, -0.8567031025886536, -1.4182636737823486, 0.9224109649658203, 0.9882210493087769, -0.9501656293869019, 0.4396805167198181, 1.5402873754501343, -0.31569382548332214, 0.038880810141563416, -1.0277788639068604, -0.04979966953396797, 0.4190645217895508, 0.1516486257314682, 1.075283408164978, 1.5368289947509766, -0.7402275204658508, -0.22768595814704895, -0.09488211572170258, 0.7065293192863464, -0.17055176198482513, -0.5008233189582825, 0.08113963901996613, 1.440895676612854, -0.9600112438201904, -0.4512940049171448, 0.2568536102771759, 1.3226277828216553], [-1.1801069974899292, -0.6796995997428894, -0.7253657579421997, -0.7635452151298523, -1.6574103832244873, 0.0376807264983654, 2.279564380645752, -2.08799409866333, 0.04932313412427902, 0.314167857170105, -0.4574904143810272, 0.8639589548110962, 0.4432869255542755, 0.5172260999679565, -0.7686432600021362, 1.049048900604248, -0.32830190658569336, 2.515613079071045, 0.5546019673347473, 0.12956503033638, 2.0545570850372314, -1.8055285215377808, 0.23111195862293243, 0.005832672584801912, 0.6589149236679077, -0.8622723817825317, -0.21258410811424255, -0.9239156246185303, -0.11116600036621094, 1.1108330488204956, -0.4301149249076843, -0.3815292716026306, 0.2521003186702728, 0.6116729974746704, 0.7084041833877563, -1.110129952430725, 0.7150770425796509, 0.20917484164237976, -0.8827512264251709, 1.012518048286438, -1.181087613105774, 0.4914892911911011, -0.9167470932006836, 0.7177589535713196, -2.614999294281006, -0.451043039560318, -0.019574785605072975, -0.040307652205228806, -1.5167073011398315, 0.794614315032959, 0.5344135761260986, 1.1222282648086548, -0.960464596748352, 0.061865128576755524, 0.6370879411697388, 0.0978793129324913, 1.054306983947754, 0.61850506067276, -0.796292245388031, 0.7245662808418274, -0.8640963435173035, 1.5059314966201782, 0.1548122763633728, 1.8584449291229248], [-0.47223398089408875, -0.9587969779968262, 0.32471075654029846, 0.3600781261920929, 0.21759697794914246, 2.2114334106445312, 0.6290370225906372, -1.3904204368591309, 0.8716539740562439, -0.13345499336719513, -1.2194160223007202, 0.9154781699180603, -1.2049425840377808, -1.9696654081344604, -0.6567336916923523, 0.7505349516868591, -0.19487011432647705, -0.970435380935669, -0.4420679211616516, -0.4861636459827423, 1.4987263679504395, -1.3743783235549927, 0.3720734715461731, 0.5076568722724915, -1.0427683591842651, -0.7643014788627625, 2.3537609577178955, 1.1975377798080444, -0.6954118013381958, -0.5836452841758728, 0.8956634402275085, 1.1820095777511597, -1.15947425365448, -1.9039100408554077, -0.363821417093277, -0.7066891193389893, -0.6354709267616272, 0.5854835510253906, -0.7891553044319153, -0.40942031145095825, -0.06446204334497452, -0.19030949473381042, 0.9565212726593018, 2.393430709838867, -0.965359628200531, 0.40946316719055176, 0.45087307691574097, 0.7144511938095093, 0.38350221514701843, -0.6625354290008545, 0.36707162857055664, -0.540052592754364, 0.00720382621511817, 0.8380831480026245, -1.5496947765350342, -1.0660455226898193, 2.116636276245117, 0.3114974796772003, 2.1135916709899902, -0.26304781436920166, -0.5904996395111084, -1.157274603843689, 2.4030609130859375, -0.4643895626068115], [-0.34037846326828003, -1.1643505096435547, -0.41661176085472107, 1.1973532438278198, 0.2593003809452057, -0.09078047424554825, -0.7356219291687012, -2.2562410831451416, 0.14389567077159882, -1.0534065961837769, -0.5635985732078552, 0.04960723593831062, 0.775377094745636, -0.48121580481529236, 0.9918762445449829, 1.9082280397415161, 0.7003567814826965, 1.125314712524414, -0.830521821975708, -0.03318123146891594, -0.4159076511859894, 0.005716696381568909, -0.0011184377362951636, -0.1715240627527237, -2.0795578956604004, -0.35095885396003723, 1.5835862159729004, -1.1027536392211914, -1.0942786931991577, 0.7534716129302979, -0.2276274710893631, 1.5794978141784668, 0.055470116436481476, 1.5237094163894653, 0.12997905910015106, 0.02015177346765995, -1.8545513153076172, 0.037858184427022934, -0.6928500533103943, 0.30853307247161865, -0.43223997950553894, -0.48217150568962097, -0.7173213958740234, 0.7787240147590637, 0.5337240099906921, -1.0684971809387207, -0.9664260745048523, -0.25863325595855713, -1.0337834358215332, 0.47283780574798584, -0.5161867141723633, 0.3763796389102936, 0.11513620615005493, 0.5427413582801819, -1.24397611618042, -0.8510127663612366, 1.212918758392334, -0.48554208874702454, 1.319474220275879, 0.8066354990005493, 0.7125495672225952, 2.126121759414673, 1.1164370775222778, -1.1550050973892212]], [[0.29379600286483765, -1.033761978149414, -0.022227633744478226, -0.41065865755081177, 0.6247479915618896, 1.1730774641036987, 0.4070069491863251, 2.183236598968506, -2.0736401081085205, -1.7485724687576294, 1.053964614868164, -0.8917016983032227, 0.4117186665534973, -0.8081931471824646, 0.1117062121629715, 0.36029383540153503, 0.5990800857543945, 0.28330445289611816, 2.2143821716308594, 2.1803410053253174, 1.1990606784820557, -2.1607701778411865, 0.22140955924987793, -0.034899987280368805, 0.14492018520832062, -0.6146076321601868, -0.2681445777416229, -1.9361324310302734, -0.2170219123363495, 0.1938774138689041, -0.8273966908454895, -0.1053445115685463, -1.592289686203003, 0.3870091736316681, -1.0036029815673828, 1.3617433309555054, -0.27784961462020874, -0.5684112906455994, 0.1380929946899414, -2.0236308574676514, -0.6723936796188354, -0.7017638683319092, 0.049650639295578, -1.0552089214324951, 1.3163049221038818, 0.08345731347799301, -0.01773170940577984, 0.6760778427124023, 1.6333268880844116, 0.9626531600952148, 1.098420262336731, 0.34171268343925476, -0.7755922079086304, 0.5383521914482117, -0.42015329003334045, 2.191377878189087, 0.0734185054898262, 0.5857579112052917, -1.1671421527862549, -0.32434937357902527, -2.261139154434204, 0.9100593328475952, 0.9073675870895386, 0.593767523765564], [1.5170814990997314, 1.5977137088775635, 1.1097825765609741, -0.23484867811203003, -0.8537803888320923, -1.5090727806091309, 0.060097791254520416, 1.4916484355926514, 0.600660502910614, -0.7815203666687012, -0.953286349773407, -1.1806412935256958, -1.3635003566741943, -1.9252345561981201, -2.101003885269165, 1.1429970264434814, 0.3558933436870575, -0.7574734091758728, 1.68100106716156, 1.4260720014572144, -0.4309532344341278, -1.3317514657974243, 0.20737788081169128, 0.2512137293815613, -0.9999034404754639, -0.5317587852478027, -0.8643555045127869, 1.2553000450134277, -0.10588293522596359, -1.244813084602356, 1.4771668910980225, -1.7421272993087769, 1.7733409404754639, -0.4615406394004822, 0.4720073342323303, -1.6159886121749878, 0.44952473044395447, -0.6486560106277466, -0.5026246905326843, -1.0521682500839233, -0.667905330657959, -0.12482386827468872, 0.38485175371170044, -1.0149205923080444, -0.7133323550224304, 0.8703773021697998, -1.7488943338394165, -0.6375049948692322, 0.953517735004425, 0.876326322555542, -0.22299864888191223, 1.300658106803894, -0.4120556116104126, 1.4720295667648315, 1.5717620849609375, -0.2936106026172638, -0.5298004150390625, -0.5232541561126709, -0.14514151215553284, -1.2038625478744507, 1.119807243347168, -0.7558993101119995, 0.4889379143714905, -1.524030327796936], [1.0423215627670288, 0.983596682548523, 1.612760066986084, 1.10123872756958, -0.6013534069061279, -1.2527459859848022, -1.0850189924240112, -0.3459410071372986, 0.0011796827893704176, 0.1718604862689972, -0.1849510222673416, 0.6654621362686157, 0.1347908228635788, 1.0559043884277344, 0.27782198786735535, -0.3478948771953583, -0.1766221523284912, 0.34102681279182434, 2.04890513420105, 1.0881190299987793, -0.6005089282989502, -0.8889688849449158, -0.8531172871589661, 0.9921748638153076, -1.294685959815979, 0.2756793797016144, -0.6255636215209961, -1.0952508449554443, -0.8821650147438049, -1.15292489528656, 0.49058687686920166, 0.7649242877960205, -2.100938558578491, -1.0556819438934326, -0.4032718539237976, -0.6197871565818787, 0.42998576164245605, -2.2840335369110107, -0.15767435729503632, 1.9707567691802979, -0.4176119565963745, 1.323085069656372, 1.1116623878479004, 0.4004004895687103, -1.0328075885772705, 0.15414002537727356, -0.3025931417942047, -2.4576430320739746, -1.052164077758789, -2.403778076171875, -1.921543836593628, -1.4094626903533936, 0.12357370555400848, 0.6071210503578186, 1.4021791219711304, -0.37923505902290344, 0.41802316904067993, 0.1711309552192688, -0.9933724403381348, -0.4886634647846222, 0.9818835258483887, -0.11560112982988358, -1.2094978094100952, -0.13872216641902924], [0.43047893047332764, -0.009995434433221817, 0.08350736647844315, -1.350221872329712, 0.39654362201690674, 0.1795332282781601, -0.4788714349269867, -1.0603113174438477, 2.2457540035247803, 2.7238283157348633, -0.7984868288040161, 0.3638075292110443, -0.658098042011261, -1.0311447381973267, 1.84538733959198, -1.2139956951141357, -0.4410371780395508, 2.029355049133301, -2.0961828231811523, 0.23334887623786926, 0.08675098419189453, 0.17490459978580475, -0.2966662347316742, -0.4971347749233246, -1.5795313119888306, -0.10364177078008652, -0.13366220891475677, 0.0733254998922348, 1.2002677917480469, -0.7830204963684082, 1.6844456195831299, 0.5677164196968079, -1.053180456161499, -0.7851506471633911, -0.7301799654960632, 0.536493718624115, -1.3137798309326172, -0.5718153715133667, -0.9210494160652161, 0.30074524879455566, -0.06194451078772545, -0.7175102829933167, -0.0889088585972786, 0.7949313521385193, -1.0996707677841187, -0.014156704768538475, 0.10816165059804916, -1.7474092245101929, -1.0791645050048828, 0.4822736382484436, -2.0128064155578613, 0.9939529895782471, -0.45927339792251587, -0.5751082897186279, 0.11580203473567963, 0.41708487272262573, -0.1476687341928482, 1.5062600374221802, -1.09579598903656, 0.9447417855262756, 0.3843725621700287, 0.6536052227020264, -0.10449638962745667, -0.03678072616457939], [-0.15297791361808777, -0.07231124490499496, 0.6085667014122009, -0.9729022979736328, 0.10013991594314575, -0.013524630106985569, -0.28456738591194153, 1.673620581626892, 0.14097757637500763, 0.80058354139328, -0.31467366218566895, 0.17179220914840698, 0.6114381551742554, 0.6152884364128113, -0.36294662952423096, -0.47064119577407837, 0.327472984790802, -0.3139301538467407, -1.02548348903656, -0.9443891048431396, -1.3890875577926636, 0.13578516244888306, 0.4432820975780487, 0.024587156251072884, -1.2562919855117798, 1.2076282501220703, -0.2640548646450043, 0.46357935667037964, 0.09078040719032288, 0.4081536829471588, -1.0134222507476807, 0.5593250393867493, -0.8187503814697266, -0.12383954226970673, -2.20935320854187, -0.04832884296774864, -0.7726936340332031, -0.04305682331323624, 1.2186870574951172, -0.908859372138977, 0.3516464829444885, 0.4598001539707184, -0.2277732789516449, 0.23840896785259247, 1.307302713394165, 1.8786073923110962, -1.1653553247451782, 0.0601806715130806, 1.4325660467147827, -0.9049167037010193, -0.7524288892745972, -0.8603733777999878, 0.9610488414764404, 0.752849280834198, -1.3058717250823975, -0.9571152329444885, -1.2885433435440063, 0.7807366847991943, -0.05771869793534279, 1.7126489877700806, 0.14574839174747467, -0.48622944951057434, -0.8541532158851624, 0.6963585019111633], [-0.8518236875534058, -0.6066771149635315, 0.717199981212616, 0.12775267660617828, 0.648613452911377, -0.9309742450714111, 1.8066269159317017, 1.2414671182632446, -1.195542812347412, 0.7617122530937195, -0.41428443789482117, 1.5195256471633911, -0.35841095447540283, 0.09266890585422516, 1.016907811164856, 0.6160759329795837, -0.25310999155044556, -1.3537373542785645, -3.026968479156494, -1.6814942359924316, -1.2253981828689575, 0.35566699504852295, 0.7345304489135742, 0.5878185033798218, 1.4356215000152588, -0.4829942584037781, -0.8440787196159363, -0.19754669070243835, 0.30870166420936584, 1.960305094718933, 0.5609182715415955, -1.165739893913269, 0.47198301553726196, 1.1746811866760254, -0.6073431968688965, 1.6317228078842163, 0.579432487487793, 2.2909903526306152, -0.36604803800582886, 0.011152789928019047, 1.2359755039215088, 0.2847987413406372, 1.2958487272262573, 0.8523054718971252, -0.9177040457725525, -0.9821090698242188, -0.4148145616054535, -1.115224003791809, -0.014860978350043297, 1.382399082183838, 0.3404565751552582, -1.6284548044204712, -2.297539710998535, -1.7221118211746216, 1.2373459339141846, 0.7104759812355042, 0.5855606198310852, 0.48615968227386475, -0.42279306054115295, 0.7639967799186707, -0.9934387803077698, -1.138543725013733, 0.7253530621528625, 0.10872791707515717], [-0.3704201281070709, 0.4526660144329071, 0.8112892508506775, 0.6678707599639893, -0.3683590292930603, 0.8009180426597595, -0.32437971234321594, -1.2416044473648071, 0.9664980173110962, -0.19214627146720886, -0.06458255648612976, -1.6147701740264893, 1.1559948921203613, -1.0771301984786987, -2.4343974590301514, -0.6781359314918518, -1.8096189498901367, -0.7759394645690918, 1.4019529819488525, 0.022584708407521248, 1.0843050479888916, -2.6290595531463623, 0.2064155787229538, -0.4309016168117523, 0.10509185492992401, 1.69313645362854, -0.3864600360393524, 0.07704392820596695, 0.7701845765113831, -0.49162524938583374, -1.3190439939498901, -0.09427855908870697, 0.8468177914619446, 0.7716147899627686, 0.36441028118133545, -1.08193838596344, -0.38615116477012634, 0.6287286281585693, 0.37753379344940186, 1.8440531492233276, -2.3213307857513428, -1.7623413801193237, -1.2128126621246338, -2.0520455837249756, 0.32394298911094666, -0.38551947474479675, -1.9082914590835571, -0.35881906747817993, 1.1972168684005737, 0.7701236009597778, -0.8030701875686646, -0.5756613612174988, -0.2631971836090088, 1.615256667137146, 0.20046596229076385, 0.13184067606925964, 1.0046559572219849, -0.030847089365124702, 1.9663515090942383, -0.25924161076545715, -0.6229419112205505, -0.17411866784095764, -1.468316674232483, -0.15605765581130981], [-1.3621492385864258, -0.18736222386360168, 1.592934250831604, -1.0766905546188354, 0.3417845368385315, 0.9328853487968445, 2.3193953037261963, 0.88956218957901, 1.080811619758606, 0.7035508155822754, -1.5790297985076904, 0.7957691550254822, 0.008448212407529354, 0.1021423190832138, 0.7547588348388672, 1.151908278465271, 2.43259859085083, 0.8742949366569519, 2.2845916748046875, 0.609062671661377, 0.4759752154350281, -0.9382205605506897, -0.9599371552467346, 0.46422433853149414, 0.833246111869812, -0.4669516384601593, -0.44256526231765747, -0.5191175937652588, -0.10294277966022491, 0.1013961061835289, 1.6599483489990234, 0.12376844137907028, 1.0350134372711182, 0.23110352456569672, -0.5256046056747437, -0.8272733092308044, -0.7213000655174255, -0.32254672050476074, 0.1536577045917511, 1.323004961013794, -0.03629593551158905, -0.7638883590698242, -1.106042742729187, -0.03014250658452511, -0.07748474925756454, 0.6707508563995361, -0.5491423010826111, 0.769892692565918, 0.43832772970199585, -0.4040118455886841, 0.38158607482910156, -0.751358151435852, 0.7642186284065247, 2.476588010787964, 0.5301169753074646, -1.2011585235595703, 1.6916335821151733, -0.07529112696647644, -2.8393423557281494, -0.06798317283391953, 0.16741269826889038, -0.5092650055885315, 0.15742306411266327, 0.5307413339614868], [-0.45807772874832153, -0.03903108090162277, -0.8601473569869995, 0.8925731182098389, -1.4902796745300293, 0.37867531180381775, -0.5116094946861267, -0.68062424659729, -0.8104346990585327, 0.23719905316829681, 0.08327005058526993, -0.8131858110427856, 1.9111049175262451, 1.5344693660736084, 0.14924375712871552, -0.5267887711524963, 0.5705775022506714, 1.2595170736312866, -0.18269850313663483, -0.07713228464126587, 1.8897123336791992, -1.3383625745773315, 0.6070398688316345, 0.1901693195104599, -1.7348350286483765, -1.5005403757095337, 0.8722686171531677, 0.3752634823322296, -0.02308719791471958, -0.289751261472702, 0.3489673435688019, 0.3391347825527191, -1.054229974746704, -0.3640972673892975, 1.040822982788086, -1.7660281658172607, -1.9305497407913208, 0.21674904227256775, -1.1094002723693848, -0.32547810673713684, 1.8084955215454102, -0.6132339239120483, -0.21445880830287933, 0.7276671528816223, -0.5195280909538269, 0.8665388226509094, 0.3445303738117218, -0.07662250846624374, -0.5968497395515442, -0.10394506901502609, 0.5197848081588745, 0.2996247708797455, -0.5613234043121338, -0.6193979382514954, 0.8766134977340698, 0.5916936993598938, -0.6324896812438965, 0.861477255821228, -2.339038610458374, -0.10808485001325607, -1.3515819311141968, 0.04124658554792404, 0.3566575348377228, -0.17075401544570923], [0.8946271538734436, 0.2589859664440155, 0.11692208796739578, -0.25455936789512634, 0.20502027869224548, -1.076751947402954, -0.44640761613845825, 0.19839708507061005, 0.3694593608379364, -0.7589526176452637, 1.1710267066955566, -1.761054277420044, 1.20622980594635, -0.40388426184654236, -0.2140917032957077, 0.0751444548368454, 0.09640590846538544, 0.07065802812576294, 1.3396223783493042, 0.22820593416690826, 0.03395051509141922, 0.20871229469776154, -0.031055081635713577, 0.00781706441193819, -1.2013823986053467, -0.544745922088623, 1.2695859670639038, -0.3392537534236908, -0.6430249810218811, 0.9171012043952942, -0.893949568271637, -1.3784921169281006, 0.5815402269363403, -0.19045524299144745, -0.13171228766441345, -0.2566499710083008, -0.27093350887298584, -1.2089943885803223, -0.04499737173318863, -1.0916670560836792, 1.5690357685089111, -1.1921157836914062, -1.5630552768707275, -0.18350531160831451, -0.28362300992012024, 0.3912138342857361, -1.7118128538131714, -1.3979778289794922, 1.3190160989761353, 0.22612638771533966, -0.45205414295196533, 1.234631896018982, 0.8114027976989746, 0.480839341878891, -0.34000083804130554, 1.1950576305389404, -1.5514708757400513, -0.40064066648483276, -0.6949754953384399, 1.0347529649734497, 0.3681574761867523, -1.2937431335449219, -0.02123730629682541, 0.8520097136497498], [-1.0737712383270264, -0.8227449059486389, -0.8980425000190735, 0.7000059485435486, 0.7447376251220703, -0.46149250864982605, -0.7782326340675354, 0.6777910590171814, 0.9602519869804382, 1.1946234703063965, 1.1997151374816895, 1.3413188457489014, 0.7035079598426819, 0.6874290108680725, 0.6002752780914307, 0.6833246350288391, 0.8459543585777283, -0.7734867334365845, 0.19039767980575562, -1.346190333366394, -1.1428046226501465, -0.06610018014907837, -0.43003031611442566, 1.4798083305358887, -1.6906641721725464, -1.7805860042572021, -1.1678386926651, 0.7438071966171265, 0.06855249404907227, 0.2381594479084015, -2.4281165599823, 0.19235600531101227, -2.0729334354400635, 0.736314594745636, 0.24030406773090363, 0.22118598222732544, 0.5718978643417358, 1.59869384765625, 1.133544921875, -0.8528141379356384, -0.4131859242916107, 0.1357838362455368, -0.9190393090248108, 0.2609397768974304, -1.3476916551589966, -0.5736337304115295, 0.9335604906082153, 0.2616707682609558, 0.5904509425163269, -1.5200273990631104, -0.3430671989917755, -1.3011000156402588, -0.35712724924087524, -0.27167895436286926, -1.8606181144714355, -0.3201800286769867, 0.1842304766178131, 0.6590456962585449, 0.5478622913360596, 1.0723118782043457, -0.19089193642139435, -1.2666757106781006, 0.6460106372833252, -1.0521548986434937], [0.26333826780319214, 1.4150030612945557, 2.391972541809082, -0.37770766019821167, -0.32568392157554626, -0.6307116746902466, 0.3179144859313965, 0.9789026379585266, -1.85917329788208, -0.2946084439754486, 0.49742355942726135, -2.2170333862304688, -0.8298041820526123, -0.14992497861385345, -0.4829234480857849, 1.3945832252502441, 0.8679708242416382, -0.21159985661506653, 1.1515133380889893, 1.9551022052764893, -0.03402986750006676, 0.19396153092384338, 0.8284330368041992, -1.8878570795059204, -0.6083704829216003, 0.6059432625770569, 0.3486049175262451, -0.7404667139053345, 0.2901788353919983, 0.3069913983345032, -1.4480695724487305, 0.4404769241809845, -1.0608876943588257, -1.5834426879882812, -1.483073353767395, 0.2006600946187973, -1.0514267683029175, -1.004096508026123, -1.5216962099075317, -1.346501111984253, 0.12435447424650192, -0.15315060317516327, -0.7858979105949402, -0.3856762945652008, 0.582298219203949, 1.2228384017944336, 1.8206369876861572, -1.5446562767028809, -0.6605222225189209, -0.07600672543048859, -0.932618260383606, 0.788916826248169, 0.38173791766166687, 0.1472935676574707, -0.8398949503898621, 1.3941789865493774, -0.25351396203041077, -1.1325029134750366, 0.9738249182701111, 0.22940292954444885, 0.12906521558761597, 0.31938233971595764, -1.0289374589920044, 1.5030204057693481], [0.854877233505249, -0.5753759145736694, -0.4162086546421051, -0.2551065683364868, -0.8400205373764038, 0.5501042008399963, -0.7265779972076416, 1.0974619388580322, 0.542427659034729, -1.7414653301239014, 0.952863872051239, -0.12106942385435104, 1.1475260257720947, -2.0924460887908936, -0.053663235157728195, 0.38089755177497864, 0.1414269506931305, 0.6163934469223022, 0.5226278305053711, 1.601729154586792, -0.32502487301826477, 0.4119957983493805, -1.557238221168518, -0.1773250848054886, -0.09302826225757599, 0.8291559219360352, -0.4367676377296448, -0.8811146020889282, 0.766208291053772, -0.02593163773417473, 1.2722718715667725, -0.26193737983703613, -1.327939510345459, -0.11844521015882492, -0.10073154419660568, -0.0541553795337677, 0.11179923266172409, 1.043758511543274, 1.0988571643829346, 0.24759405851364136, 0.2310238629579544, 0.7990807294845581, -0.07761833071708679, -0.8836459517478943, -0.620472252368927, -0.8767330050468445, -1.1371934413909912, -1.4449152946472168, -0.020573463290929794, -1.2133811712265015, -1.139507532119751, -1.6064497232437134, -0.3857325315475464, 0.9327858090400696, 1.2051106691360474, -0.693229615688324, -1.4153796434402466, 0.044169262051582336, 1.4147495031356812, 1.8105323314666748, 0.7959572076797485, 0.32315710186958313, 1.887172818183899, -0.230772003531456], [-0.6164863705635071, 0.1258050799369812, -1.02329683303833, 1.0465123653411865, -1.692522406578064, -2.033257246017456, -1.2646284103393555, 1.3956564664840698, 0.027536028996109962, -0.1643088459968567, -0.4911349415779114, -1.279335856437683, 0.23333293199539185, -0.2703148126602173, -0.39281708002090454, 1.8968439102172852, -1.0760177373886108, 0.24435217678546906, 1.2752745151519775, 0.25280094146728516, 0.9808491468429565, -0.42698419094085693, -1.1659730672836304, -0.20958812534809113, 0.6127137541770935, -2.084510087966919, -1.0329614877700806, -0.317966103553772, -0.6239050626754761, 0.6183537840843201, -0.38537171483039856, 0.972830057144165, 0.022487269714474678, -0.15422126650810242, -1.1317192316055298, 0.8260193467140198, 0.5978918075561523, -0.6525976657867432, 1.2700093984603882, 0.37337496876716614, 0.053180795162916183, 0.060180019587278366, -1.6027874946594238, 1.3190213441848755, 1.2467446327209473, -2.1695072650909424, -0.8307439684867859, 0.16637763381004333, 0.5371795296669006, -0.6317961812019348, 0.43277212977409363, 1.3110865354537964, 0.41952526569366455, -2.2577059268951416, -0.26846975088119507, 0.5106538534164429, -0.761814534664154, 1.84787917137146, 1.9579678773880005, -1.5360289812088013, -0.1424192488193512, -0.02178564853966236, -0.1627911925315857, 0.26084548234939575], [0.43661314249038696, -0.2414318472146988, 0.14561425149440765, -1.308881163597107, -0.39816734194755554, -0.5462417006492615, -1.2170478105545044, -0.5744243860244751, -1.1998035907745361, -1.083511233329773, 0.08235546946525574, 0.3518218994140625, -0.8028209209442139, 0.3973790407180786, -0.9316569566726685, 1.1817201375961304, 1.371740460395813, 0.19642938673496246, 1.4336248636245728, 0.6106500029563904, 0.2498319000005722, -0.11763354390859604, -0.09181436896324158, -2.257608413696289, -1.2694711685180664, 0.10023458302021027, 0.28564557433128357, 0.41778290271759033, 0.7471086978912354, 0.5741215944290161, -0.558364987373352, 0.3388823866844177, 2.1512410640716553, -1.155625581741333, 1.397856593132019, -1.6322695016860962, -1.332436203956604, -1.2439290285110474, -0.3471486568450928, 0.4241544306278229, -1.1775776147842407, 2.4698407649993896, 1.1551275253295898, 0.837478756904602, -0.5046268105506897, -0.5168555378913879, 1.1213879585266113, -1.223412275314331, 0.4167473316192627, 0.33169975876808167, -1.7414416074752808, -0.4354480803012848, 0.4401790499687195, 2.595454216003418, -1.1938410997390747, -0.00423900643363595, -1.210082769393921, 0.5995676517486572, -0.21577821671962738, 0.23916643857955933, -0.903936505317688, -0.8932706713676453, -0.799383819103241, 1.526442050933838], [1.244177222251892, -1.5609349012374878, -0.014700428582727909, 0.023847470059990883, -0.7378594279289246, -1.0237842798233032, 0.7357694506645203, -0.3197748064994812, 0.2770327031612396, -1.5225008726119995, -0.40665575861930847, -0.6651167869567871, 0.5243309140205383, 1.2279689311981201, 0.16878584027290344, 1.5288140773773193, -1.4004722833633423, 0.04172775521874428, -1.0862528085708618, 0.7808581590652466, 1.0357142686843872, 0.7411890625953674, 1.7030127048492432, -0.1182507798075676, 0.9688611030578613, 1.673435091972351, -0.405636191368103, 1.0566306114196777, 0.36889180541038513, 0.4935324490070343, 0.5098717212677002, 1.0606403350830078, 1.0427896976470947, -1.5892866849899292, 1.805802345275879, -1.0805822610855103, -0.38528329133987427, 0.09025704860687256, 1.3948156833648682, 0.4826594293117523, 0.44564205408096313, 0.1202598288655281, -0.18483620882034302, -0.6667836308479309, 0.6521203517913818, 0.1918163299560547, 0.5847765803337097, -0.12214060127735138, -0.7577099204063416, 0.41970741748809814, -0.43191829323768616, 0.6332747340202332, 0.1298779547214508, 1.0364412069320679, -0.3560875356197357, 2.389754295349121, 0.9094513058662415, -1.6111561059951782, -0.28110387921333313, -2.1357333660125732, -2.2019500732421875, 0.46554338932037354, 0.49156731367111206, -0.4512139558792114]], [[-0.8108217716217041, -0.5062617063522339, -1.0191096067428589, 0.59858638048172, -0.4157624840736389, -0.6956205368041992, 1.396060585975647, -1.0650922060012817, 0.21211354434490204, -0.7994386553764343, 0.7341753244400024, 0.4526841938495636, 0.38891711831092834, -1.0876144170761108, 1.4239203929901123, -0.4551355838775635, -1.352705478668213, -0.6776034832000732, -0.16494300961494446, -0.1512230932712555, 0.08915002644062042, 0.7560480237007141, -1.5829291343688965, 1.616010069847107, -0.011021710000932217, 1.1062259674072266, -0.07007952779531479, -1.7896252870559692, 1.8437044620513916, -1.5055036544799805, -0.89080810546875, 0.5754106044769287, -0.10043148696422577, 1.0737128257751465, -1.350972056388855, -1.184163212776184, 1.9360729455947876, -2.0015106201171875, 0.9859004616737366, -0.8949152827262878, -0.45217764377593994, -1.8409160375595093, -0.19167517125606537, 1.7310082912445068, -0.3193243741989136, 0.21251854300498962, 0.4368169605731964, 0.9217638969421387, -1.4728364944458008, 0.507264256477356, 0.616753339767456, 0.3888758420944214, -0.25431886315345764, -0.48181280493736267, 0.7579705715179443, -0.8228191137313843, 1.3423587083816528, -0.6673862338066101, -1.1833293437957764, 0.7355154752731323, -1.25493586063385, -0.6617661118507385, -0.13775944709777832, 1.6465412378311157], [1.1086344718933105, -0.2845178544521332, -0.3986859619617462, 0.6232139468193054, -1.3754031658172607, 0.03349268063902855, -0.4000994861125946, -0.053989894688129425, -0.7085622549057007, -0.12360750883817673, -0.9853197932243347, 0.8417776823043823, 1.3485510349273682, -1.431314468383789, 1.2043448686599731, 0.73621666431427, 1.1114400625228882, 0.5671207308769226, -1.7491868734359741, 0.8423221707344055, -0.5486358404159546, 0.14014016091823578, 0.22438858449459076, 0.7205467224121094, 0.5544707775115967, -0.4356054663658142, -0.0576472133398056, 1.2470625638961792, 1.1676708459854126, -1.2839834690093994, 0.09018780291080475, -1.0105431079864502, -1.022587776184082, 0.331484854221344, -0.7385722994804382, 0.22200126945972443, -0.8096156120300293, 0.2533736824989319, -0.5065194368362427, -1.4563326835632324, -0.7167063355445862, -1.7625041007995605, 0.08424802869558334, -0.8493521213531494, -1.571010708808899, -1.0613067150115967, 0.34622669219970703, -0.8297327160835266, -0.9065073728561401, 1.4550541639328003, 0.2673538625240326, -0.7574788331985474, 0.42398038506507874, 0.7026776075363159, 1.5293742418289185, -0.8264188170433044, -0.813681960105896, 0.3269205093383789, -1.3046878576278687, -0.48533517122268677, -0.13662704825401306, 0.06345263123512268, 0.17131257057189941, -0.34133824706077576], [0.9721643328666687, -0.6435630321502686, 0.6199789643287659, -1.6274977922439575, 1.0149072408676147, 0.8372604846954346, -1.8610605001449585, 0.5495816469192505, -0.8884925246238708, -0.21403588354587555, -0.804940402507782, -1.4797781705856323, -0.428843230009079, -1.3692868947982788, 0.48183736205101013, 0.5053368210792542, -0.590229332447052, 0.1544562727212906, 0.31811705231666565, -0.2567838132381439, -0.6960275173187256, 0.735867440700531, -0.84342360496521, -0.5542234182357788, 1.2679697275161743, 1.0353416204452515, 0.18649360537528992, 1.091156005859375, 1.6059225797653198, -0.4139987528324127, -1.801291584968567, -1.2492954730987549, -0.10723190009593964, -0.015550727024674416, -0.2737917900085449, -0.03227326646447182, -0.5588713884353638, 0.46886777877807617, 1.6281543970108032, -0.09358493983745575, -1.0045937299728394, 1.3088067770004272, 0.32764577865600586, 1.7854644060134888, 0.6224036812782288, -0.2167828530073166, 1.3093630075454712, -0.6276587247848511, -0.24879446625709534, 0.3533645570278168, 0.8978461027145386, 0.24126039445400238, 0.3311309516429901, 0.23221491277217865, 1.1817430257797241, -1.410203218460083, -0.21040961146354675, -2.100414752960205, 1.6477969884872437, 0.2634263038635254, 0.6352719068527222, 0.6500139236450195, -0.3793676793575287, 0.9674933552742004], [-1.1185302734375, 0.7619010806083679, -0.4845370352268219, 0.9541173577308655, 0.2357150912284851, 0.6733672618865967, -0.7703367471694946, 0.9386944770812988, -0.854851245880127, -0.30533111095428467, 0.5723596811294556, -2.7687931060791016, -1.2814419269561768, 1.63728666305542, -0.11295377463102341, -1.5150829553604126, -0.09186920523643494, -1.2359378337860107, -1.1091368198394775, -0.8008785247802734, -0.34197157621383667, -0.45730727910995483, 0.5741875171661377, 1.0225508213043213, -0.5599552392959595, -1.093846082687378, -0.6004961133003235, -0.34509825706481934, -1.3132585287094116, 0.5741591453552246, 0.8411993384361267, 0.3125031590461731, 0.3710804879665375, 0.19067712128162384, 0.41628867387771606, 0.8165492415428162, -0.8855357766151428, 1.4841724634170532, 0.33906781673431396, 0.414714515209198, 0.4036620557308197, 0.8839324712753296, -0.9824337959289551, 0.8768293261528015, 1.2248480319976807, 0.39880621433258057, 0.25157779455184937, -0.6131690144538879, 1.0392584800720215, 0.7007963061332703, -1.0319058895111084, 0.7059895396232605, -1.304284930229187, -1.4575823545455933, -0.018608370795845985, 0.8541086912155151, -0.8423318266868591, -2.786341667175293, 1.0846461057662964, 0.19264040887355804, 0.6939143538475037, -0.2830562889575958, 1.4822059869766235, -0.09847814589738846], [-0.6087254881858826, -0.09137792885303497, -1.0100418329238892, -0.38932499289512634, 0.21737711131572723, 1.958367109298706, -0.5376480221748352, -0.12458726018667221, 0.48774734139442444, 0.23015885055065155, -1.0408315658569336, 2.674163818359375, 0.20042039453983307, -0.2256462723016739, -1.2940661907196045, -1.1815496683120728, -0.6397846341133118, 0.2885441482067108, 1.418820858001709, 2.315915107727051, 0.5627363920211792, 0.12119091302156448, -0.7284409999847412, 2.344334840774536, -1.9800766706466675, 0.5416402816772461, -0.8678253889083862, 0.5947590470314026, -0.5099436044692993, 1.2499473094940186, 1.9902608394622803, -2.903489828109741, -0.1864517331123352, -0.048054344952106476, -1.973419189453125, -1.1759026050567627, 3.353095293045044, -1.1276881694793701, 0.4941856265068054, -0.6724860668182373, -0.9970352053642273, -1.4313839673995972, 1.4527788162231445, 0.4913719892501831, -0.2348436415195465, -0.5298936367034912, -2.748934030532837, -1.3890035152435303, -1.7417536973953247, -0.052121520042419434, -0.1596732884645462, -0.2985828220844269, 0.5797184705734253, 0.07622235268354416, 0.021560601890087128, -1.14341402053833, -0.90522700548172, -0.5982983708381653, 0.12528346478939056, -2.391662836074829, -0.6270266175270081, 0.3778892457485199, -0.3081560730934143, -0.7684164643287659], [-0.6480324268341064, 0.14340229332447052, 0.5827521085739136, 0.4499548375606537, -0.2740565538406372, -1.1242120265960693, 1.383880615234375, -0.11668703705072403, 0.8061861395835876, -2.4570014476776123, -1.0363898277282715, -1.2391822338104248, -0.18538959324359894, -0.8871958255767822, 1.464768886566162, -0.4467197060585022, 0.23148462176322937, -0.8096698522567749, 1.590395212173462, -0.5243474245071411, 1.122497797012329, -0.5516127347946167, 0.41438159346580505, -0.635617196559906, -1.0521037578582764, 0.6004607081413269, 0.16561995446681976, 1.2286452054977417, -0.3186356723308563, -0.9575469493865967, 0.6135205626487732, 1.729921817779541, 0.39283907413482666, 0.7785207629203796, 1.7607648372650146, 0.9529813528060913, -0.19015395641326904, -0.400452584028244, 0.37136444449424744, -0.3160744309425354, -0.6501811742782593, 0.38876232504844666, 0.48783302307128906, 1.4051870107650757, -1.0159709453582764, 0.7794016599655151, -0.5804306268692017, 0.6144244074821472, -0.2817080318927765, 1.1749849319458008, -0.43217435479164124, -0.4831508696079254, -0.9244054555892944, -1.1304317712783813, -1.7134371995925903, 0.7816197276115417, 1.5412397384643555, -0.9526974558830261, -1.017469882965088, -0.19211457669734955, 1.8124632835388184, -0.8436775803565979, 2.388305187225342, 0.9559189677238464], [-1.1842514276504517, 0.10136216878890991, -1.9799911975860596, 0.3560991585254669, 0.8275438547134399, -0.6547729969024658, 0.30031704902648926, 0.5976992249488831, -0.9831905364990234, -0.08753586560487747, -0.2945746183395386, -1.749563455581665, 0.22413575649261475, 0.3996920883655548, 0.7545836567878723, -0.12686507403850555, 0.16610322892665863, -0.09753858298063278, -0.8736509084701538, 0.6878985166549683, 0.7266064286231995, -0.7901099324226379, -1.0999011993408203, 1.5736931562423706, 0.3323396146297455, -0.5207479596138, -0.15501351654529572, -0.48719900846481323, 0.5512398481369019, -0.7832298278808594, 1.0113649368286133, 1.2540888786315918, -1.4310388565063477, -0.2760842740535736, -0.49102792143821716, -0.08046353608369827, 0.055476605892181396, 0.6373344659805298, -0.4502585530281067, 0.14656361937522888, 0.5339588522911072, 0.46038818359375, -1.0991755723953247, -0.31760865449905396, 0.9191721081733704, 0.43303579092025757, -1.1865882873535156, 1.5717754364013672, -0.06773082166910172, 0.49580445885658264, -0.0810810849070549, 1.2671180963516235, 0.9382593631744385, 0.3124609589576721, 0.15127642452716827, 0.5498105883598328, 0.5616100430488586, 0.7367844581604004, -0.9171274900436401, -1.3158997297286987, 0.10808456689119339, 0.4097357988357544, -0.8293839693069458, -0.5659608244895935], [-1.6037253141403198, -0.8259798288345337, 1.310821771621704, -0.826941192150116, 0.6040141582489014, -0.40524429082870483, 0.2597787082195282, -0.8576651811599731, -1.2342439889907837, 0.3181736469268799, -0.4543704390525818, 0.013740234076976776, -1.6280326843261719, 0.5742838978767395, 0.28023040294647217, -0.42335838079452515, -0.9883494973182678, -1.1675443649291992, 1.4210543632507324, -0.46860963106155396, -0.46819302439689636, 0.11694495379924774, -1.613903284072876, 0.6496901512145996, -0.3553488850593567, -0.3957936763763428, -0.43632572889328003, 0.20334628224372864, -0.05946258082985878, -0.6851460337638855, -0.1556297093629837, -0.8703758120536804, 1.3356060981750488, -0.5914511680603027, -0.43153539299964905, 0.5324031710624695, 1.1065006256103516, 0.4634275436401367, 0.4286671578884125, -1.2501957416534424, -1.2474853992462158, 0.10427656024694443, -2.092494249343872, -0.02886008657515049, -0.1412764936685562, -0.7792001366615295, -1.5680887699127197, 0.2567473351955414, 1.9466561079025269, -0.3231861889362335, -1.0205491781234741, 0.8446351289749146, 1.435383915901184, 0.8953104615211487, -1.56322181224823, -0.6558473110198975, 0.7088415026664734, 1.9035191535949707, -0.9363176226615906, 1.2015407085418701, 0.8421400189399719, -0.9926269054412842, 0.29781574010849, -0.9132325053215027], [-0.12686608731746674, 1.6987214088439941, -0.22526657581329346, 0.6320297122001648, 1.0237222909927368, 1.0412917137145996, 0.22462831437587738, -0.08174294978380203, 0.9721484184265137, -0.5429718494415283, 0.48603397607803345, -1.4667561054229736, 0.06857725232839584, -0.382470041513443, 0.31633689999580383, -0.5830503702163696, -0.08900438994169235, 1.2956290245056152, -0.9924084544181824, -0.38895225524902344, 0.07091198116540909, -2.5187294483184814, 0.2273178994655609, 1.717881441116333, 0.19095386564731598, -0.047827672213315964, -2.147381544113159, -0.9804943203926086, 0.06645741313695908, -0.5024636387825012, -1.2268580198287964, -0.46935394406318665, -0.06403493881225586, -1.6687862873077393, -1.273267388343811, -0.7613643407821655, 1.0705313682556152, -0.6813364624977112, -2.561659812927246, -2.8396525382995605, -3.3065364360809326, -0.23663851618766785, -0.5373290181159973, 1.7519769668579102, 0.1109187975525856, -0.39982739090919495, 1.0091158151626587, -0.10340479761362076, 0.048912566155195236, -0.8502042889595032, 0.06826183199882507, -0.5092670321464539, 0.6710600256919861, -1.2668678760528564, 0.3941337466239929, 0.18251748383045197, -0.8148084282875061, -1.966932773590088, -0.7623695135116577, -0.5820465087890625, -0.6685307025909424, -0.8748776316642761, 0.9843993186950684, -1.015629529953003], [0.5403697490692139, 1.7715132236480713, 1.3404792547225952, 0.6260108947753906, -0.565959095954895, 1.1941218376159668, 0.5288747549057007, 0.8729874491691589, -0.038458529859781265, -0.15122991800308228, -0.3218701183795929, 0.6495691537857056, 0.4360849857330322, -1.1165674924850464, -0.7047882676124573, -0.11000417172908783, -0.05542590469121933, 0.4497741162776947, -0.11568254977464676, 0.1332564800977707, 2.06733775138855, 1.6627885103225708, -1.2368907928466797, -0.36926743388175964, 0.514597475528717, 0.47980910539627075, -2.0229427814483643, 0.21975252032279968, 1.7734614610671997, 0.8768730759620667, -0.07300413399934769, -1.1835501194000244, 0.4953381419181824, -0.40962302684783936, 0.19756382703781128, 0.501523494720459, 0.3480397164821625, 0.3653925955295563, 0.043569695204496384, -1.2995288372039795, -0.042209066450595856, -0.15591368079185486, -0.06503307074308395, -0.6612363457679749, -1.4183028936386108, 1.126502275466919, 0.3242468237876892, 0.6553371548652649, 0.04047120362520218, 0.14413122832775116, -0.02477244846522808, 0.2572459578514099, -0.854138970375061, 0.3430667817592621, -1.196600317955017, -0.6552523970603943, -0.7731720805168152, 1.5246881246566772, -1.0476070642471313, -0.24573221802711487, 1.4857852458953857, 0.060995347797870636, 1.6705434322357178, 0.7932440638542175], [-0.8761656880378723, 1.434401512145996, 0.7470182776451111, -1.2284992933273315, -1.9238088130950928, 0.8795962333679199, -0.976458728313446, -0.11215785890817642, 0.1313599795103073, 0.8297985196113586, -0.260751873254776, -1.2971296310424805, -1.0822908878326416, -0.07420941442251205, 0.8923530578613281, -0.5604590773582458, -1.1927109956741333, -1.7129440307617188, 1.831176996231079, -0.8945080637931824, -0.8180778622627258, -0.4469364583492279, 0.5830597281455994, 0.3479999303817749, -1.3774000406265259, -1.9189939498901367, -0.7222215533256531, -0.05845959112048149, -0.7012341618537903, 0.3483732044696808, -0.5442785620689392, 1.7226295471191406, -0.5497955083847046, -0.7877218127250671, 1.0027556419372559, 0.11624602973461151, -1.4857133626937866, -0.5350291132926941, 0.03638904169201851, -3.158050060272217, -1.058684229850769, 0.45977839827537537, 0.5582239627838135, 0.14955785870552063, -1.4337059259414673, -0.3900126814842224, -0.9962124824523926, -0.19722333550453186, -0.42111000418663025, 0.4703396260738373, -1.2974553108215332, -0.6769660115242004, 0.026296434924006462, 0.544815719127655, -1.352258563041687, -0.1882728487253189, 2.3508832454681396, -0.02998480387032032, -1.2426518201828003, 0.8667811155319214, 0.4152630865573883, 0.15602776408195496, -0.13315492868423462, 0.7979995012283325], [-1.1120878458023071, 1.0234758853912354, 0.8432161211967468, 0.9551160335540771, 0.7853630781173706, 0.8530339002609253, 0.06845501065254211, -0.3855501115322113, 0.2677370607852936, -0.648896336555481, 1.297553539276123, 1.6835461854934692, -1.2748066186904907, 0.7495880722999573, 1.7296981811523438, -1.1880521774291992, 1.0081596374511719, -0.3490184247493744, 0.9228730797767639, -1.1342352628707886, 0.48851102590560913, -1.442508339881897, 0.9037883281707764, 2.0066051483154297, -0.6801832318305969, -0.38057902455329895, -0.49868831038475037, -0.17262284457683563, -0.683193027973175, -1.297165036201477, 0.19644126296043396, -0.13660310208797455, -0.3935316801071167, -1.6235889196395874, -0.45297664403915405, 0.3673171103000641, 1.8695319890975952, 1.178902506828308, -2.3433215618133545, 0.7012152671813965, -0.10217411071062088, -0.020414341241121292, -0.8455340266227722, 0.19163087010383606, 0.14699847996234894, -0.1956997662782669, -2.022486686706543, 0.3505612313747406, -1.0416916608810425, -0.11160045117139816, -0.04914337024092674, -1.6019082069396973, -0.08691605925559998, -1.3505754470825195, -0.01697418838739395, -1.133294701576233, 0.00045579299330711365, -0.5610684156417847, 1.1101741790771484, -0.7011329531669617, -0.1697668433189392, -1.3695828914642334, -0.4972992539405823, -0.4174233078956604], [-2.1533427238464355, 0.2780894637107849, 0.9293559193611145, 1.3983900547027588, 0.6385074257850647, -0.39436987042427063, -0.5610167980194092, -0.15230222046375275, 0.7444414496421814, 2.2802646160125732, 1.127907156944275, 0.825374186038971, 1.3582744598388672, -0.06785822659730911, 0.4491623640060425, 0.6151761412620544, 0.10047239810228348, -2.2148709297180176, -1.2812169790267944, -0.2120654433965683, 0.06590209156274796, 0.09917894750833511, -0.0850449651479721, 1.230357050895691, 0.3859768211841583, -0.8417347073554993, 0.08405528962612152, -0.6689485311508179, -0.25106489658355713, 1.8801192045211792, 0.946444571018219, -0.596778929233551, 1.0175116062164307, 2.5795109272003174, -0.41062262654304504, -0.672478199005127, 0.8153509497642517, -0.9929235577583313, -1.177337646484375, -0.247749462723732, -2.275952100753784, -1.2247092723846436, -0.4233790934085846, -0.44109007716178894, -0.5768764019012451, 2.1136474609375, -1.7541754245758057, 0.6883673667907715, 0.1823730319738388, 0.5209910869598389, -1.3642973899841309, -0.11465055495500565, -0.29235997796058655, 0.937405526638031, -0.43567267060279846, -0.9311327934265137, -0.3519708216190338, 0.4003846347332001, 0.3131067454814911, 0.802956223487854, 0.702176034450531, 0.614444375038147, 0.979072630405426, 1.7529891729354858], [0.6656903624534607, 0.025498678907752037, 0.2606123685836792, -0.24471785128116608, 0.9466665387153625, 0.16219455003738403, 1.6588948965072632, -0.42755892872810364, 1.5932048559188843, 1.3428199291229248, -0.2257310003042221, -0.09820124506950378, -0.2141498625278473, 0.10055015236139297, 0.36284077167510986, -1.0083260536193848, 0.777600884437561, -0.3705707788467407, 0.21185439825057983, -0.08160558342933655, 1.4065001010894775, 0.004135641735047102, -0.8304876089096069, 1.259500503540039, -0.12625224888324738, -0.0349152572453022, 0.3104855716228485, -0.26645445823669434, 0.4018051028251648, -0.879584014415741, -0.9232315421104431, 1.6896276473999023, 0.8986268043518066, 0.15712012350559235, -1.2849053144454956, -0.5697464346885681, 0.9877853393554688, -1.1755801439285278, -0.8608309030532837, -0.21503467857837677, -0.12041272968053818, 1.7380025386810303, -0.040064696222543716, 0.4080647826194763, 0.5259643197059631, 0.19861535727977753, -0.8528923392295837, 0.6070464253425598, -1.2807505130767822, -1.0176126956939697, -0.603259265422821, 0.9015291333198547, -0.30807051062583923, -1.9597833156585693, 1.4398270845413208, 0.5164029002189636, 0.28486624360084534, 0.7694216966629028, -1.0336265563964844, -1.3554480075836182, 0.5565475821495056, -0.11413849145174026, 0.06618990749120712, -1.4211928844451904], [-0.11838334053754807, 0.9249824285507202, -1.030845284461975, -0.4469893276691437, 0.20604687929153442, 0.37310898303985596, 0.36888599395751953, 1.8132249116897583, 0.6609694361686707, 1.131730079650879, -0.4450690746307373, -0.11175280064344406, -1.4361861944198608, 0.5117630362510681, 1.1012203693389893, -0.9735636711120605, 1.8503384590148926, 0.202415332198143, -0.34419843554496765, -1.5976817607879639, -1.0718026161193848, -0.06990957260131836, -1.032163143157959, -0.6320728063583374, 0.11301866173744202, -0.9421790242195129, -0.6846460700035095, 0.11562883108854294, -1.5112251043319702, -2.0922369956970215, 1.1970323324203491, -0.8123003244400024, 0.3279111385345459, -0.9558683037757874, -0.17359785735607147, 1.2204068899154663, 0.5559923648834229, -1.2449339628219604, -1.0444884300231934, -0.5058140754699707, -1.0600193738937378, 0.036862414330244064, -1.1629917621612549, -0.9148513078689575, 1.2269625663757324, 0.478664368391037, -1.6032990217208862, 0.9571318626403809, -1.0785127878189087, 0.3971026539802551, 0.1932820826768875, -1.174519419670105, 0.727136492729187, -0.9564926028251648, 0.005550199653953314, 0.017953073605895042, -0.8708004951477051, -0.8198776245117188, -0.7947651147842407, -1.2349377870559692, 0.9023256301879883, 0.3733726441860199, -0.034994032233953476, 0.20747129619121552], [-1.156217098236084, 1.5066142082214355, -0.08646980673074722, -0.5119755268096924, -1.4071986675262451, -0.08943633735179901, -0.0920383557677269, 0.011723233386874199, 1.4497960805892944, -0.7417081594467163, 2.3944895267486572, 1.971122145652771, 0.6181578636169434, -1.0305594205856323, -0.9335946440696716, -0.3519677221775055, 0.12784826755523682, -1.5964021682739258, -0.18257708847522736, -0.33858686685562134, 0.4184616506099701, 1.2082096338272095, -0.060660719871520996, 0.36649075150489807, -1.0595704317092896, -1.339651346206665, -0.06064685806632042, 0.1853167712688446, -1.0733637809753418, 0.10292116552591324, 1.022308588027954, -2.2884304523468018, -0.733651340007782, 1.6000657081604004, 0.0018690330907702446, 0.9229584336280823, -0.8816967606544495, -0.34499141573905945, -0.43603116273880005, 0.548014223575592, 0.6120191812515259, -0.20438003540039062, 0.5002683997154236, 0.3662923574447632, -0.17953874170780182, -0.8299857378005981, 0.9311074614524841, -2.1409120559692383, -1.1713253259658813, 0.012508250772953033, 0.5350683331489563, 2.8615152835845947, -0.055437151342630386, -0.12653864920139313, -0.8437842726707458, -0.04963362589478493, -0.1317971795797348, -1.1506539583206177, -0.7455146312713623, 0.3558819591999054, -0.7702377438545227, 0.32355642318725586, -0.11948097497224808, 0.017812497913837433]], [[1.254486322402954, 0.09234260022640228, -0.23623865842819214, 0.0022732627112418413, 1.89565908908844, -0.4733649790287018, 0.6138757467269897, -1.0313355922698975, -0.7504520416259766, 0.40530768036842346, 0.19281764328479767, -0.6141998767852783, -0.6645469069480896, -1.425936222076416, 0.5992810726165771, -0.6684064269065857, -1.4814926385879517, 0.9466201663017273, 0.49561673402786255, -0.15506060421466827, -0.9554294347763062, -0.7528620958328247, 1.527987003326416, 0.2353033572435379, -0.3253372311592102, 1.055663824081421, 1.47284734249115, -0.5754345059394836, 0.3777865171432495, -1.2737423181533813, -0.009960454888641834, -0.5545715689659119, -0.8826829195022583, -0.5261927247047424, 0.6532803177833557, -0.9481467604637146, -0.008594760671257973, -0.27969396114349365, -0.5765721797943115, -1.9334310293197632, -0.20899051427841187, -0.5055789947509766, 0.3021266758441925, 1.068616509437561, -0.4239632487297058, -0.1900772750377655, -0.1137954443693161, -0.008864933624863625, -0.418031245470047, 2.640353202819824, 1.6757533550262451, -0.8910405039787292, -0.8331870436668396, 1.5326248407363892, -0.9930164813995361, 0.3353513777256012, -1.8990983963012695, 1.895397424697876, -0.1892758160829544, 0.5618941783905029, -0.4052532911300659, 1.082837700843811, -0.38296130299568176, 1.3361860513687134], [1.051093578338623, 0.3400823175907135, 0.29405975341796875, 0.8602290749549866, 0.9941594004631042, -1.4064619541168213, 0.8040024638175964, -0.49689966440200806, -0.1018182560801506, -0.28630563616752625, 1.6546876430511475, 0.9666355848312378, -0.8947629332542419, -0.6771166920661926, 0.6950398683547974, 1.6066093444824219, -0.9614517688751221, -1.8675062656402588, 0.43080976605415344, -1.573994517326355, 0.3559541702270508, 0.6806436777114868, 1.010437250137329, -1.1801533699035645, 0.9609536528587341, 2.303304433822632, 1.3759400844573975, 0.5768452286720276, 0.19439971446990967, -2.9273335933685303, -0.33838868141174316, -0.556199312210083, 0.7343938946723938, -1.045401692390442, 2.350724220275879, 0.9320806264877319, 1.5602083206176758, 1.586914300918579, 2.8894875049591064, -1.1677616834640503, 0.5944746732711792, -0.506796658039093, -0.010494403541088104, 0.01685679331421852, -1.2710251808166504, 0.6837683320045471, 1.4069912433624268, -1.2845102548599243, 0.10437534749507904, 0.7366681694984436, -0.0881725400686264, 0.8360627889633179, 0.4748457074165344, 1.0667074918746948, -0.2399580329656601, 0.5680872201919556, -0.0597250834107399, 1.262528896331787, 2.0253729820251465, -0.698793888092041, -0.18513137102127075, -0.042568061500787735, 0.20413580536842346, 1.375461220741272], [-0.08011199533939362, -0.6390484571456909, -1.2734531164169312, 0.19904173910617828, 1.5093164443969727, -0.22149065136909485, -1.5829826593399048, 0.026317445561289787, -0.04026658460497856, 0.24408580362796783, 0.20365332067012787, 2.1303939819335938, -0.3344924747943878, 0.20206992328166962, -0.40246084332466125, -2.1725685596466064, -0.9953146576881409, -0.7031156420707703, 0.12412043660879135, -0.09429235756397247, -0.5726944804191589, 0.18957899510860443, -0.9796783924102783, -1.220031976699829, 0.48600271344184875, -1.0998215675354004, 0.7450482845306396, 0.28250938653945923, -1.7386428117752075, 0.04159202054142952, 1.640862226486206, 0.5279664993286133, -0.6000640988349915, 0.8453179597854614, -0.8179137110710144, -2.022470712661743, -1.4800978899002075, 1.1471139192581177, 1.390639305114746, -0.9529756903648376, -0.5501164197921753, -0.8563632369041443, 1.7011433839797974, -0.4013345241546631, 1.056357741355896, -0.7855393886566162, 0.16394451260566711, -1.4057338237762451, -1.6731528043746948, -0.861010730266571, 0.7589048147201538, -1.0416964292526245, -0.3087104856967926, 1.1154849529266357, 0.4760136008262634, 0.8846256136894226, 0.8713136911392212, 0.9426550269126892, -0.42536500096321106, 1.6586655378341675, -0.7063673138618469, 0.6656580567359924, -0.3989678621292114, 1.3697690963745117], [-0.05133689567446709, 0.4599374532699585, -0.7738202810287476, 0.4507061243057251, -0.4334682524204254, 0.4907001852989197, -0.4825705289840698, 0.40518853068351746, -1.0063841342926025, -1.9737944602966309, 1.3013256788253784, 1.2324798107147217, -1.1296066045761108, 1.0009828805923462, 1.3801240921020508, -0.6957630515098572, 0.1588834524154663, 0.2669041156768799, 0.3122367560863495, -0.18114538490772247, 0.35541632771492004, 1.3557676076889038, 1.9633965492248535, -1.639173150062561, -1.3289260864257812, -0.08373536169528961, -0.6738297939300537, 0.4782455861568451, -0.6913794875144958, 2.2787723541259766, -0.24938742816448212, -0.19702908396720886, -0.1454777866601944, 0.6471844911575317, -0.042814306914806366, 0.36853039264678955, 0.5364194512367249, -0.04788120836019516, 1.0303760766983032, -0.5617873072624207, 0.6837433576583862, -0.5400437116622925, -0.9546551704406738, 0.7485347986221313, -0.14855964481830597, 0.4369436502456665, 2.366891622543335, 0.18670734763145447, -0.732357382774353, 0.7006641030311584, -0.7018889784812927, 0.044227030128240585, -2.6940081119537354, 0.9231756329536438, -1.3635894060134888, -1.013353705406189, 0.3768293261528015, -0.5636170506477356, 0.5570676326751709, -0.9837700128555298, 0.6596919894218445, 0.5487020611763, -0.5188086032867432, 1.0505313873291016], [1.7328282594680786, -0.8591100573539734, 0.16382086277008057, 1.02608060836792, -0.857750654220581, 0.01695091463625431, 0.9175348281860352, -0.36336854100227356, -0.373641699552536, 1.519412636756897, -0.20587623119354248, -0.6461209654808044, 0.30445072054862976, 2.021820545196533, 0.05109196528792381, 0.5199925303459167, 0.3594219982624054, -0.7722760438919067, -0.5767354369163513, 1.1106635332107544, 0.29588884115219116, 1.0441148281097412, -0.01896689273416996, -0.701978862285614, 1.6045583486557007, 1.681281328201294, 0.4296543002128601, -0.34889936447143555, 0.41567757725715637, -0.08320286870002747, 0.5600736737251282, -0.06555716693401337, -0.9219725131988525, -0.44945451617240906, 1.4181963205337524, 0.07600300759077072, -0.7953028678894043, -0.09073822945356369, 0.41024935245513916, 0.38550329208374023, 0.8241957426071167, 0.9747191071510315, 0.33275020122528076, 0.30090534687042236, -0.7184060215950012, -1.8277779817581177, 1.2284889221191406, -0.6062048673629761, -0.10514110326766968, -0.32993951439857483, 0.24823863804340363, -1.8963385820388794, -0.4035358726978302, -0.6818127036094666, -1.2908868789672852, -0.19044053554534912, 0.5790769457817078, -0.5030381083488464, -0.8667823076248169, 0.06504832208156586, 1.3898332118988037, 1.5390851497650146, 0.2902604043483734, -0.29333144426345825], [0.3969423174858093, -0.2297351062297821, 0.6674098372459412, -0.2296467125415802, -1.004698395729065, 1.3327476978302002, -0.6955792903900146, -0.2973088026046753, -0.1445763260126114, -0.1359063982963562, -0.6372644305229187, 1.155012845993042, 0.5856202244758606, -1.7717849016189575, 0.44061699509620667, 1.310434103012085, 0.9296697974205017, -0.013109183870255947, -0.9374477863311768, 0.17329655587673187, -0.618979275226593, 0.7090063691139221, 1.7014459371566772, -2.729278326034546, 0.6168829798698425, 0.8529706001281738, 0.10805223882198334, -0.5566285848617554, 0.309123694896698, -0.8407822847366333, 2.2328009605407715, 1.4967283010482788, -2.1728620529174805, -0.6143198609352112, 1.5849697589874268, 1.4876859188079834, -1.074368953704834, 1.3596889972686768, 0.6220659017562866, -0.002222247887402773, 1.887699007987976, 0.4388011395931244, -0.3217565417289734, -1.22614324092865, -0.9415585994720459, 0.5252453088760376, -0.11975947767496109, 0.5567493438720703, 0.879229724407196, -1.181449055671692, 0.42171695828437805, 0.31977540254592896, 1.1751965284347534, 0.21884162724018097, 0.8756730556488037, 0.18733927607536316, -2.1001925468444824, 0.5433297157287598, 1.6686519384384155, 0.4932098090648651, -0.6612712740898132, 0.7601012587547302, -1.3917354345321655, -1.424584984779358], [0.5900898575782776, -0.036063481122255325, -0.08774916082620621, 0.7565192580223083, -0.8146538734436035, 1.7768582105636597, 2.0284385681152344, -0.09019559621810913, -0.2586174011230469, -0.7357426285743713, 0.5027526617050171, 1.0485484600067139, -0.36406758427619934, -0.016205357387661934, -1.8689731359481812, 0.45781585574150085, 0.3931941092014313, 0.009473816491663456, -0.2040414661169052, 2.7073147296905518, -1.320896029472351, 0.5991062521934509, 0.5439258217811584, -0.5045467019081116, -0.6319723129272461, 1.281265377998352, -1.2828916311264038, 0.36223000288009644, -1.035904049873352, -0.38049882650375366, -0.8599603176116943, 1.5454280376434326, -2.6143643856048584, -0.6392390727996826, 1.3356754779815674, -1.1579034328460693, -0.0012400925625115633, 1.0903840065002441, 2.1889283657073975, 0.5332349538803101, -0.7501007914543152, -0.3563743531703949, -1.8078855276107788, -0.4164092242717743, -0.6006200313568115, 0.5730782747268677, 0.034618597477674484, -1.8098491430282593, -0.35296958684921265, 1.0967013835906982, -0.3509405553340912, 0.9433785676956177, -0.12023957073688507, 0.27587321400642395, 0.30662181973457336, -0.10602748394012451, -1.6579396724700928, 1.2300201654434204, -0.3705892264842987, -0.3439942002296448, -0.3494061827659607, 0.25394973158836365, 0.5740320682525635, -0.9738729000091553], [-0.6617287397384644, 1.419137716293335, -1.6583671569824219, 0.757900595664978, -0.4410153031349182, -0.4485146105289459, 1.192224383354187, -1.3696157932281494, -0.1555447280406952, 1.9712462425231934, -0.6800957322120667, -0.7842507362365723, -0.6022161841392517, 2.0734875202178955, 1.1394822597503662, 1.267562985420227, -0.5406359434127808, 0.3467550575733185, 0.20780010521411896, 0.08632142841815948, -0.6551579833030701, -1.0006450414657593, 0.43547049164772034, 0.4006590247154236, -0.8212209343910217, -0.43081313371658325, 0.6496454477310181, 1.1129027605056763, -1.5023505687713623, -1.298144817352295, 0.1290665715932846, -1.2241308689117432, -0.6136072278022766, -0.324585497379303, 0.23025673627853394, -2.3550639152526855, -1.2292860746383667, 1.3266342878341675, 0.6628735661506653, 0.31042271852493286, -0.2140570729970932, 0.17298947274684906, 0.6411919593811035, -1.0679529905319214, 0.7342174053192139, -0.11358863115310669, -0.01682031899690628, 1.373594880104065, -0.12456347048282623, 0.38982337713241577, -0.8585120439529419, -1.797844648361206, 0.3549498915672302, -0.08086742460727692, 1.5454070568084717, -0.020721156150102615, -0.08078524470329285, 0.6450209021568298, 1.0976808071136475, 1.4000024795532227, -0.2051495462656021, -0.7326139807701111, -0.5148844718933105, 1.164844036102295], [-1.1035879850387573, -0.8977574706077576, 0.07443231344223022, -0.8403390049934387, 0.1543159782886505, -0.05557269975543022, 2.406235933303833, -1.5643484592437744, -0.44164684414863586, 0.9369006156921387, -1.0498065948486328, 1.4818105697631836, -0.6984427571296692, -1.1291102170944214, 0.4727080464363098, 0.3051779568195343, 0.38344278931617737, -1.4008910655975342, 0.46076104044914246, -0.6872954964637756, -1.5473839044570923, 0.14238132536411285, 1.0986188650131226, 0.6259728670120239, -2.030421495437622, 1.0342715978622437, -0.6445484161376953, 0.08141064643859863, -0.5733956694602966, -0.9434992671012878, 0.21986964344978333, -0.19142337143421173, -1.4195626974105835, 1.0663944482803345, 2.1819093227386475, -0.325400710105896, -0.051696132868528366, 1.1284198760986328, 0.18715178966522217, 0.3754727840423584, -1.7014411687850952, -1.6795580387115479, -1.332910180091858, 0.10510756820440292, 1.2279325723648071, -0.27337104082107544, 0.134952113032341, 1.1781647205352783, 0.4312902092933655, 0.04474771022796631, -0.07339678704738617, 0.989196240901947, 0.2005060613155365, 1.1094529628753662, -0.6407033205032349, -0.732649028301239, 0.7427920699119568, 0.5483769178390503, -0.03122003935277462, -0.16030998528003693, 0.2227596491575241, -0.13946785032749176, -1.349852204322815, 1.2738895416259766], [1.5178894996643066, 0.8526248335838318, -0.02024882286787033, -0.028470490127801895, 0.8561342358589172, -0.005369804799556732, 1.5824239253997803, 0.18311017751693726, 0.2934928834438324, -0.4799100458621979, -1.766545295715332, 1.1870999336242676, -1.1552674770355225, 0.5170052647590637, 1.9760956764221191, 0.3467336595058441, -0.21423275768756866, -0.9902697801589966, 0.1854947805404663, -1.0880377292633057, 0.20408016443252563, -0.425953209400177, -0.29888153076171875, 1.0295583009719849, -0.06896030157804489, -0.32088226079940796, -0.4115464389324188, -0.30475643277168274, 0.6725583672523499, -0.5479241609573364, -0.746456503868103, 1.3547990322113037, 1.3744926452636719, -0.05150971561670303, 0.5477932095527649, 0.5533096194267273, 0.017471060156822205, 0.2966715097427368, -0.19525626301765442, -0.8190592527389526, -0.09622941166162491, -0.02955729141831398, 0.4041266143321991, 0.008385873399674892, -0.119330994784832, 0.02926531806588173, -1.062907338142395, 0.8145274519920349, -0.1350114643573761, 0.1935250610113144, -0.8593724370002747, 1.9583882093429565, -1.6604657173156738, 0.5690098404884338, -0.6615419387817383, 0.01357161346822977, -0.7689329385757446, 0.1389205902814865, -1.4909082651138306, -1.8161593675613403, 0.01803554594516754, 0.4097355902194977, 0.6786646842956543, 0.5613609552383423], [-0.2842416763305664, -0.2757139205932617, 0.16297060251235962, -1.2657184600830078, 1.42116117477417, -0.10733456909656525, -1.3671163320541382, 1.8563836812973022, 0.42229294776916504, 0.8371570110321045, -0.0348573736846447, 1.077755331993103, -0.03605236858129501, 0.24263033270835876, 0.5682504177093506, -0.7212643623352051, -0.5867546200752258, 1.5793696641921997, 0.673146665096283, -0.18436671793460846, -0.1443823277950287, 1.6477750539779663, -1.161731243133545, 2.5764379501342773, 0.8581480979919434, 0.6012915968894958, -0.08143725246191025, -0.7542558908462524, 1.7173258066177368, 0.569124162197113, 0.2389427125453949, -0.4374189078807831, -0.2546921670436859, 0.005248097702860832, 0.10576923936605453, 0.45317336916923523, -0.7456457614898682, -1.161926507949829, 0.9156950116157532, 1.1355931758880615, -0.4564942419528961, 1.1495084762573242, 0.11332248151302338, -0.7149174809455872, 1.879555344581604, 1.97102689743042, 0.5867334008216858, 0.06134738400578499, -0.10014092177152634, 0.784268319606781, -0.5227826237678528, -0.4481256604194641, -1.1917757987976074, -0.22230233252048492, 1.1099984645843506, -0.15226411819458008, -0.858089804649353, 0.014218173921108246, 0.9385407567024231, -0.9361422061920166, 0.7839502096176147, -1.0716632604599, -1.0214333534240723, 0.08049675077199936], [1.930928111076355, -0.28655993938446045, -1.074705719947815, 0.026180265471339226, 1.6166929006576538, 0.8723781704902649, -0.5454716086387634, -0.7357169389724731, 0.22672700881958008, -0.3166390359401703, 0.43341290950775146, -0.4340493977069855, 0.5577810406684875, -0.36785560846328735, -1.137155532836914, 0.1837969720363617, 1.0158584117889404, 0.08294340968132019, -1.1734592914581299, -0.4040977656841278, 1.6384060382843018, -1.2882870435714722, 0.26256704330444336, -1.8330214023590088, -0.9077407717704773, 0.6912578344345093, 0.44079697132110596, 0.0626671239733696, -0.14830847084522247, -0.24312925338745117, -0.8641943335533142, -0.028921185061335564, 0.3239539563655853, 0.21150928735733032, -1.1148195266723633, 0.9734564423561096, -1.3247804641723633, 0.7705498337745667, -0.36596807837486267, -0.3651595711708069, -1.9348198175430298, 0.03342161327600479, 0.9929704666137695, 0.04216454178094864, -0.49434277415275574, -0.5137221217155457, -0.3308502435684204, -1.0935019254684448, 0.5900043845176697, 0.3376641273498535, 1.7697583436965942, 0.8968605399131775, -0.832683801651001, -1.4468683004379272, 1.4414862394332886, 0.2673836350440979, -0.4308736026287079, -1.1064413785934448, 0.9582619071006775, 1.5154838562011719, 0.546137809753418, -1.3205689191818237, 0.9438039660453796, 0.07939749956130981], [-1.2554677724838257, -0.2714936435222626, 0.23767568171024323, -0.3950667977333069, -1.9573999643325806, -0.4982661008834839, -0.8151910901069641, 0.8241630792617798, -0.3092364966869354, -2.267634630203247, -0.7605307102203369, 0.9292632341384888, 0.5152273774147034, -1.2669649124145508, -0.9888591170310974, -0.6695317625999451, 1.3451124429702759, 0.3685375452041626, -1.2309516668319702, -0.2308957725763321, -1.11933434009552, 0.10492845624685287, -0.3450464606285095, -0.7203315496444702, 0.7949134707450867, -0.27062538266181946, -1.7617638111114502, 0.6949855089187622, -0.8708959221839905, -0.7491686940193176, 0.1101294755935669, -1.101638913154602, 0.05905924364924431, -2.5742650032043457, -0.9221000671386719, 2.0456416606903076, 0.2703128457069397, 0.4960068166255951, -0.16666658222675323, -0.06744701415300369, -0.9222181439399719, -0.1688355952501297, -1.9821383953094482, 0.5716105699539185, -0.38511282205581665, 0.5018463730812073, 0.02891436032950878, -0.5977847576141357, -0.13469235599040985, -1.2661432027816772, 0.7218348979949951, 0.6053847670555115, 0.1387947052717209, -1.297383427619934, 1.0112591981887817, -0.7945533990859985, -0.32754021883010864, -0.5278482437133789, 0.12232554703950882, 0.2144654095172882, -0.7176479697227478, 1.4139032363891602, -1.1421078443527222, 0.7400149703025818], [-0.49746766686439514, -0.6187182068824768, 0.30169764161109924, 0.5091976523399353, 0.7509012818336487, -0.9584770798683167, -0.29538488388061523, 0.3994344472885132, -0.24135783314704895, -0.33086490631103516, 0.6990493535995483, 0.7554682493209839, 0.15268419682979584, -1.3364534378051758, -1.5728411674499512, -0.8130767941474915, 0.405691921710968, 0.5439302921295166, 2.95871901512146, 1.0739291906356812, -0.027142774313688278, -0.4459717869758606, -0.3778790533542633, 0.7117545008659363, -0.23226284980773926, -0.22598621249198914, -0.12651561200618744, 0.7133755683898926, -0.3240633010864258, -1.841986894607544, 1.7038300037384033, -1.5193142890930176, 0.570756733417511, 0.8149915337562561, 1.0883798599243164, -0.16047236323356628, -0.0819181427359581, 0.14829590916633606, -0.03670230135321617, -0.3303065001964569, -0.6352334022521973, 1.9373103380203247, -0.9124931693077087, -1.926808476448059, -2.032707929611206, -0.09354919195175171, -0.546625018119812, -2.1128151416778564, -1.237405776977539, -0.7983731627464294, -1.3533908128738403, -0.5610562562942505, 0.5268997550010681, 1.6116293668746948, -0.7418019771575928, -0.1959889978170395, -0.5806713700294495, -0.2986430823802948, 1.8360776901245117, -1.0807591676712036, 1.6639589071273804, 0.03620420768857002, -0.2325177937746048, 0.629131019115448], [0.39671790599823, -1.1543651819229126, 0.10580408573150635, -0.4617915153503418, 1.966755986213684, -0.9983991384506226, -0.2449445128440857, 0.9999015927314758, 0.990925133228302, 1.6138345003128052, 0.2734874188899994, 2.659619092941284, 0.3170609176158905, 1.2267723083496094, 0.6903663873672485, -1.2727234363555908, -0.5474052429199219, -1.1593492031097412, -0.1788429468870163, 0.9574762582778931, 0.2341923713684082, 0.6849752068519592, -0.40806183218955994, 0.4765303134918213, -0.7336244583129883, 0.6966680884361267, -0.8019495606422424, 0.2671279013156891, 1.2109185457229614, -1.277207612991333, -0.18860094249248505, 0.26447793841362, -0.6861206293106079, -0.5145655274391174, -0.589881420135498, -0.8915826678276062, -0.41016626358032227, -1.9141225814819336, 0.5848650932312012, -0.7720810174942017, 0.4000387191772461, 0.678636372089386, -0.8738877177238464, -0.973773717880249, 0.6089136600494385, -0.4085315465927124, 0.21050405502319336, 0.19894099235534668, 1.9623091220855713, -0.8330149054527283, 0.1304449439048767, -0.3889632523059845, 1.3562932014465332, -2.1128182411193848, -1.6504998207092285, -0.1330500692129135, -0.5233599543571472, 0.8940584063529968, 1.9150326251983643, 0.027773914858698845, 0.318155974149704, -0.9195266962051392, -1.6093964576721191, 0.40098655223846436], [0.2673974931240082, 1.3915255069732666, 0.7407464981079102, -0.4037449359893799, 0.3286682665348053, 1.7988651990890503, -0.05268010124564171, 0.9868413805961609, 0.05931561440229416, -1.1796917915344238, 1.5384982824325562, 0.6749634146690369, -0.36273112893104553, 1.1220369338989258, 1.544262170791626, -0.7048447728157043, 0.750203013420105, 0.8485108613967896, -0.39557746052742004, -0.13409222662448883, -0.4428395628929138, 1.5343388319015503, 0.7289606332778931, 0.4502353370189667, -0.45953696966171265, 0.8283787369728088, 0.2888459861278534, 0.758395254611969, -1.2632538080215454, -0.999126672744751, 0.6450923681259155, -0.16759490966796875, 0.369845986366272, -0.602825403213501, -0.6920320391654968, 0.8196582198143005, -0.5216243267059326, -1.5369179248809814, -0.43851473927497864, -0.9260963797569275, -0.6519713401794434, 0.3839642405509949, -0.6281265616416931, -0.5288437008857727, 1.451055645942688, 0.18507277965545654, -0.13895836472511292, 0.6585877537727356, 1.4199190139770508, 1.3154404163360596, 0.8737612366676331, 1.3185343742370605, 0.6920467019081116, 1.029231071472168, 1.6560025215148926, -1.2752419710159302, 0.014149273745715618, -0.9259061813354492, 0.3197450339794159, 0.9105768203735352, 1.0447567701339722, 0.8399017453193665, 1.0138843059539795, 0.8497070074081421]]]], "dtype": "float32"}, {"__nd__": [1, 8, 16], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[0.17727810144424438, -1.8182085752487183, -1.4374210834503174, 1.678144931793213, -1.0526779890060425], [-0.8458672761917114, -0.968479573726654, 0.012484724633395672, -1.2236356735229492, -0.03376154229044914], [-0.9888051152229309, 0.12838155031204224, 1.3349615335464478, -2.6634562015533447, -1.4068419933319092], [1.618716835975647, 0.22434107959270477, 0.5732019543647766, -0.5358597636222839, -0.6221776008605957], [0.4900573194026947, -0.6940538883209229, -0.8334124088287354, -0.25798651576042175, -0.14200690388679504], [-0.405674546957016, 0.4812791645526886, 0.3616921305656433, 0.39554363489151, -0.23921525478363037], [-1.2549593448638916, 1.5606955289840698, 2.3275301456451416, -0.9583427906036377, 0.6478042602539062], [-1.3484452962875366, -0.7437373995780945, -2.010759115219116, 0.9155169129371643, -0.3865358531475067], [0.5516882538795471, 1.0503448247909546, -0.48606300354003906, 0.5499415397644043, 0.7945635914802551], [1.434138536453247, 0.006106901913881302, -0.5421494841575623, 0.3288377523422241, -0.44993942975997925]], [[0.5477780699729919, 1.1584008932113647, -1.4964944124221802, -1.1144682168960571, -0.3911689519882202], [-0.32180255651474, -0.015371703542768955, 0.17988859117031097, -0.838455080986023, -0.985135555267334], [-1.4875128269195557, 1.2465163469314575, 0.372885525226593, 0.0012930368538945913, -0.5124452114105225], [-2.20465350151062, -0.48662111163139343, 0.6122915744781494, 0.7284331321716309, 0.1743437796831131], [-0.9212288856506348, -0.520760715007782, 0.4606911242008209, -0.1258200705051422, -0.24819472432136536], [-0.11391239613294601, -0.569734513759613, -0.34366869926452637, -0.8028050065040588, 0.3569880723953247], [-1.1731090545654297, -1.8446120023727417, 1.6473617553710938, 0.4442524313926697, -1.9230103492736816], [0.6976547241210938, 0.3641758859157562, -0.12046130001544952, -0.5660470128059387, 0.8612756729125977], [0.3687923550605774, 1.040311574935913, -0.7587447166442871, -0.27124887704849243, 0.1703954041004181], [-0.3515922725200653, 0.3708038330078125, -0.7154426574707031, -0.38649311661720276, 0.07755286991596222]]], [[[0.6814311146736145, -0.14496226608753204, -0.11524906754493713, 0.037602297961711884, 0.35373327136039734], [-0.26361292600631714, 0.6790767908096313, 0.3450893759727478, 0.292332261800766, -1.3875118494033813], [0.3218492269515991, 0.048533711582422256, 0.017930833622813225, -0.2235507369041443, -0.7854865789413452], [1.2760798931121826, 0.4267428517341614, -1.7227989435195923, 1.9273624420166016, -0.6215112209320068], [-0.8419966101646423, -1.3863202333450317, 1.7175085544586182, -1.707938313484192, -2.370115041732788], [0.40933334827423096, 0.5693991184234619, -0.8606952428817749, -0.17025992274284363, -0.171012282371521], [-0.1536436229944229, -0.49616241455078125, -1.2615588903427124, -0.912377119064331, -1.1017409563064575], [-1.59791898727417, -2.0760579109191895, -0.2654980421066284, -0.2329295426607132, 0.24096767604351044], [0.5951599478721619, -0.10764236748218536, 0.8009335398674011, -0.6860808730125427, 1.0090947151184082], [-0.13829925656318665, 1.302764892578125, -0.7943537831306458, -0.44558918476104736, -0.29636991024017334]], [[0.45918214321136475, -0.18720076978206635, 0.2289111316204071, 0.6913467645645142, -0.3165162205696106], [0.26153936982154846, -0.41348034143447876, -0.34797459840774536, 0.3424426019191742, 0.8169965147972107], [0.6332646012306213, 0.16442151367664337, 1.8514659404754639, 1.4545966386795044, 1.0551050901412964], [-0.9892287850379944, -0.1641509234905243, 0.6826279759407043, -0.5258460640907288, 0.8997617959976196], [2.0278642177581787, -0.7307366132736206, 0.7351508140563965, -0.4252094328403473, 0.6618818044662476], [-1.3903381824493408, 1.367071270942688, 0.64183509349823, -0.37777289748191833, -0.7703218460083008], [-0.223825141787529, -0.5723553895950317, -0.18147073686122894, 1.045082688331604, -1.929278016090393], [-0.2474316507577896, -0.5652621984481812, 1.7713851928710938, 0.4497365355491638, 0.269153892993927], [-1.501359462738037, -0.2190125286579132, 0.12943816184997559, 1.445806622505188, 0.1436576396226883], [-0.23554639518260956, 0.24768130481243134, 2.2471609115600586, -1.3483282327651978, -2.0204579830169678]]]], "dtype": "float32"}, {"__nd__": [[[[0.9480609893798828, 0.2287970632314682, -2.012953996658325, -0.024489592760801315, -1.197503685951233], [-1.6594475507736206, 1.9251536130905151, 1.8222543001174927, -0.27397462725639343, -0.01758819818496704], [0.6671196818351746, -1.4212297201156616, 0.784805178642273, -0.11470801383256912, 0.6883690357208252], [0.4704943895339966, -0.5563449859619141, -0.8128380179405212, -0.7319852113723755, 0.07098668813705444], [0.3830561339855194, 1.753658652305603, 0.8494983315467834, 1.1132537126541138, -0.2452573925256729], [1.34271240234375, 1.3709324598312378, -1.2741252183914185, -0.03894229978322983, 0.8531354069709778], [-0.3671441674232483, -1.4067829847335815, -0.9061658978462219, -1.5893816947937012, -1.504780888557434], [0.7931442856788635, 0.3435366749763489, 1.4021013975143433, -0.9433460235595703, 0.19559277594089508], [0.7931838631629944, 0.4596008062362671, -0.5433712005615234, 0.9411459565162659, -0.384891152381897], [-0.36653539538383484, 1.137444019317627, -0.21046888828277588, 0.8396810293197632, 1.5859283208847046]], [[-1.8872296810150146, 2.2490508556365967, -2.3927407264709473, -0.0685480609536171, -0.9472435712814331], [0.016013918444514275, 0.412514328956604, 1.4746742248535156, 1.0367205142974854, -1.7262197732925415], [0.458685964345932, -0.26671093702316284, 0.3884653151035309, 0.35453712940216064, -0.11585266143083572], [-0.8408119678497314, 0.5803640484809875, 1.935826301574707, 0.1666433960199356, -0.3676814138889313], [-0.6362338066101074, 1.2780996561050415, -0.8852354288101196, 1.8303314447402954, 0.17804011702537537], [-2.0668604373931885, 1.1335030794143677, -0.8185120820999146, -1.3609956502914429, -1.6148028373718262], [-2.0480778217315674, 0.1807461678981781, -0.19996915757656097, 1.5522176027297974, -1.7192336320877075], [-0.6058452129364014, 1.66104257106781, -0.21871332824230194, -0.13633233308792114, -1.1891893148422241], [-0.23350121080875397, 0.14217673242092133, 1.0427463054656982, 0.5440263152122498, -0.5709211230278015], [1.2023868560791016, -1.0719317197799683, 1.148423671722412, -2.470545768737793, 1.6034835577011108]]], [[[-0.7314071655273438, -1.3615024089813232, -1.2098251581192017, 0.21046780049800873, 0.2963881492614746], [-0.39890795946121216, 0.5365934371948242, 1.2459909915924072, 0.035926759243011475, 0.7931755781173706], [1.027029275894165, 0.26090720295906067, -0.46707889437675476, 0.7592133283615112, -0.7169386148452759], [1.0120071172714233, -0.6766987442970276, 0.6609066128730774, -1.1671074628829956, -1.125087857246399], [-0.21895021200180054, 2.792917251586914, 0.09008839726448059, 0.7068283557891846, -0.874869704246521], [0.8755368590354919, 0.41181039810180664, 2.376220226287842, -0.7940782308578491, -2.0048208236694336], [-0.15821488201618195, 1.454566240310669, -1.6604501008987427, -0.018274910748004913, 0.7698227167129517], [-0.28448453545570374, 0.04156943038105965, 0.10168658941984177, 0.38306185603141785, 1.2710591554641724], [-0.4765428602695465, 0.14382438361644745, -2.021972417831421, -0.3983151316642761, -0.3167426288127899], [-0.933758020401001, 1.3952914476394653, 0.6352418065071106, 0.08079241216182709, -0.2621016204357147]], [[-0.2127629965543747, -0.9670915603637695, -0.2810947299003601, 1.2533007860183716, 0.8626034259796143], [-3.133700132369995, -0.2882281243801117, -0.7599200010299683, 1.07877516746521, 0.6547843813896179], [-0.4456079304218292, -0.13348250091075897, -0.500577986240387, -0.867441713809967, 0.7384610772132874], [0.272602915763855, 0.7808023691177368, 0.45980674028396606, 0.14509467780590057, 0.05388648435473442], [0.1285974383354187, 2.0233545303344727, 1.3586080074310303, 0.8645979166030884, 0.8392256498336792], [0.4472369849681854, -1.7880430221557617, -1.3160597085952759, 0.5598303079605103, -0.5731616616249084], [0.05700382590293884, 0.8413258790969849, 0.43287211656570435, -1.161295771598816, 1.2820168733596802], [0.04659273847937584, -1.4127001762390137, -1.4433835744857788, -0.9020309448242188, 1.1348381042480469], [0.28637760877609253, 0.08188751339912415, 0.4322303235530853, 0.9197400808334351, -0.29576125741004944], [-0.6327240467071533, 1.0320489406585693, -0.307350218296051, 0.16092908382415771, 1.1447004079818726]]]], "dtype": "float32"}, {"__nd__": [[[[-2.452347993850708, -0.16795845329761505, -0.4645891785621643, 1.5092641115188599, 0.9964154958724976], [-0.9409003853797913, -1.1013193130493164, 0.9396400451660156, -0.6714072823524475, -0.6021928787231445], [0.7268873453140259, 0.13749662041664124, -0.3847658932209015, -2.144035577774048, -0.005853018257766962], [-1.1853147745132446, 0.9529017210006714, 1.9767018556594849, -0.5937128663063049, 0.8713198900222778]], [[0.04864155873656273, 0.16826185584068298, 0.4368758201599121, -1.0053027868270874, -0.8157366514205933], [-1.8950450420379639, -0.7845675945281982, -0.4539409279823303, -1.0161786079406738, -1.2561750411987305], [-2.3653624057769775, -1.0074793100357056, -0.01539335586130619, -0.4205716848373413, -0.24806466698646545], [1.0272444486618042, -0.8172637224197388, -0.7561119794845581, -2.4247212409973145, 0.05478478595614433]]], [[[-0.47481343150138855, 0.7721560001373291, -1.5062980651855469, -1.5350008010864258, 1.4349360466003418], [1.1382204294204712, 0.6192311644554138, 0.13430696725845337, 0.3934754729270935, 0.2341698706150055], [1.0834308862686157, 0.5054759979248047, -1.1782420873641968, -0.7089162468910217, -0.8118776679039001], [0.48589175939559937, -0.35165777802467346, -0.4529326558113098, 0.14774927496910095, -1.0618085861206055]], [[0.6603496074676514, -0.20398136973381042, 2.2117502689361572, -0.3803738057613373, 0.27662402391433716], [-0.518243134021759, -0.4501582384109497, -0.8187305331230164, -0.5024767518043518, -0.05429841950535774], [0.4235815405845642, -0.7799751162528992, -0.3494569957256317, 0.8072296977043152, -2.5290679931640625], [1.9078317880630493, 0.08490978926420212, -0.8360302448272705, 0.895765483379364, 0.6815470457077026]]]], "dtype": "float32"}, {"__nd__": [[[[0.17814256250858307, -0.27840402722358704, 0.7415899038314819, -0.48934030532836914, 0.19071871042251587], [1.821691632270813, -0.4494190216064453, 0.1786351501941681, -1.9977781772613525, 0.7125278115272522], [1.9837530851364136, -0.38311755657196045, -1.804628610610962, -0.2125563621520996, -1.6439738273620605], [0.372898668050766, 1.2346107959747314, 0.8407620787620544, -1.9741735458374023, 2.269620895385742]], [[-0.8925716280937195, 0.1363976001739502, -0.041629306972026825, 0.9385299682617188, 0.11465196311473846], [-1.1424397230148315, -1.9755349159240723, 0.7632198929786682, 1.7368512153625488, 1.5461575984954834], [-1.0612214803695679, -0.23758378624916077, 1.9129836559295654, 0.39654794335365295, -1.097297191619873], [0.19036641716957092, 0.29260456562042236, 1.1051254272460938, -0.8356581926345825, 0.27297064661979675]]], [[[0.6082843542098999, 0.4481736719608307, -0.5000821948051453, -0.6818879246711731, 1.1973731517791748], [1.2906732559204102, 0.8209097385406494, 0.8980866074562073, -1.2387152910232544, 1.412272334098816], [-0.601768970489502, -0.44018569588661194, -0.8141152262687683, -1.8893481492996216, 0.8354376554489136], [0.205281600356102, -0.9354938268661499, 0.4435957372188568, 1.2571161985397339, -0.4373035728931427]], [[1.5475643873214722, 0.6876397132873535, -1.3480664491653442, 0.8322606682777405, -0.3094295263290405], [0.29030588269233704, -1.750824213027954, 0.6169719099998474, 1.0465784072875977, -1.2813528776168823], [0.182408407330513, 0.6537931561470032, 0.6570174098014832, -0.5917274355888367, -1.8351609706878662], [-0.2355767786502838, -1.356040120124817, -1.1268818378448486, -0.1736147105693817, -0.5588804483413696]]]], "dtype": "float32"}, {"__nd__": [6, 0], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[0.17727810144424438, -1.8182085752487183, -1.4374210834503174, 1.678144931793213, -1.0526779890060425], [-0.8458672761917114, -0.968479573726654, 0.012484724633395672, -1.2236356735229492, -0.03376154229044914], [-0.9888051152229309, 0.12838155031204224, 1.3349615335464478, -2.6634562015533447, -1.4068419933319092], [1.618716835975647, 0.22434107959270477, 0.5732019543647766, -0.5358597636222839, -0.6221776008605957], [0.4900573194026947, -0.6940538883209229, -0.8334124088287354, -0.25798651576042175, -0.14200690388679504], [-0.405674546957016, 0.4812791645526886, 0.3616921305656433, 0.39554363489151, -0.23921525478363037], [-2.452347993850708, -0.16795845329761505, -0.4645891785621643, 1.5092641115188599, 0.9964154958724976], [-0.9409003853797913, -1.1013193130493164, 0.9396400451660156, -0.6714072823524475, -0.6021928787231445], [0.7268873453140259, 0.13749662041664124, -0.3847658932209015, -2.144035577774048, -0.005853018257766962], [-1.1853147745132446, 0.9529017210006714, 1.9767018556594849, -0.5937128663063049, 0.8713198900222778]], [[0.5477780699729919, 1.1584008932113647, -1.4964944124221802, -1.1144682168960571, -0.3911689519882202], [-0.32180255651474, -0.015371703542768955, 0.17988859117031097, -0.838455080986023, -0.985135555267334], [-1.4875128269195557, 1.2465163469314575, 0.372885525226593, 0.0012930368538945913, -0.5124452114105225], [-2.20465350151062, -0.48662111163139343, 0.6122915744781494, 0.7284331321716309, 0.1743437796831131], [-0.9212288856506348, -0.520760715007782, 0.4606911242008209, -0.1258200705051422, -0.24819472432136536], [-0.11391239613294601, -0.569734513759613, -0.34366869926452637, -0.8028050065040588, 0.3569880723953247], [0.04864155873656273, 0.16826185584068298, 0.4368758201599121, -1.0053027868270874, -0.8157366514205933], [-1.8950450420379639, -0.7845675945281982, -0.4539409279823303, -1.0161786079406738, -1.2561750411987305], [-2.3653624057769775, -1.0074793100357056, -0.01539335586130619, -0.4205716848373413, -0.24806466698646545], [1.0272444486618042, -0.8172637224197388, -0.7561119794845581, -2.4247212409973145, 0.05478478595614433]]], [[[-0.47481343150138855, 0.7721560001373291, -1.5062980651855469, -1.5350008010864258, 1.4349360466003418], [1.1382204294204712, 0.6192311644554138, 0.13430696725845337, 0.3934754729270935, 0.2341698706150055], [1.0834308862686157, 0.5054759979248047, -1.1782420873641968, -0.7089162468910217, -0.8118776679039001], [0.48589175939559937, -0.35165777802467346, -0.4529326558113098, 0.14774927496910095, -1.0618085861206055], [-0.8419966101646423, -1.3863202333450317, 1.7175085544586182, -1.707938313484192, -2.370115041732788], [0.40933334827423096, 0.5693991184234619, -0.8606952428817749, -0.17025992274284363, -0.171012282371521], [-0.1536436229944229, -0.49616241455078125, -1.2615588903427124, -0.912377119064331, -1.1017409563064575], [-1.59791898727417, -2.0760579109191895, -0.2654980421066284, -0.2329295426607132, 0.24096767604351044], [0.5951599478721619, -0.10764236748218536, 0.8009335398674011, -0.6860808730125427, 1.0090947151184082], [-0.13829925656318665, 1.302764892578125, -0.7943537831306458, -0.44558918476104736, -0.29636991024017334]], [[0.6603496074676514, -0.20398136973381042, 2.2117502689361572, -0.3803738057613373, 0.27662402391433716], [-0.518243134021759, -0.4501582384109497, -0.8187305331230164, -0.5024767518043518, -0.05429841950535774], [0.4235815405845642, -0.7799751162528992, -0.3494569957256317, 0.8072296977043152, -2.5290679931640625], [1.9078317880630493, 0.08490978926420212, -0.8360302448272705, 0.895765483379364, 0.6815470457077026], [2.0278642177581787, -0.7307366132736206, 0.7351508140563965, -0.4252094328403473, 0.6618818044662476], [-1.3903381824493408, 1.367071270942688, 0.64183509349823, -0.37777289748191833, -0.7703218460083008], [-0.223825141787529, -0.5723553895950317, -0.18147073686122894, 1.045082688331604, -1.929278016090393], [-0.2474316507577896, -0.5652621984481812, 1.7713851928710938, 0.4497365355491638, 0.269153892993927], [-1.501359462738037, -0.2190125286579132, 0.12943816184997559, 1.445806622505188, 0.1436576396226883], [-0.23554639518260956, 0.24768130481243134, 2.2471609115600586, -1.3483282327651978, -2.0204579830169678]]]], "dtype": "float32"}, {"__nd__": [[[[0.9480609893798828, 0.2287970632314682, -2.012953996658325, -0.024489592760801315, -1.197503685951233], [-1.6594475507736206, 1.9251536130905151, 1.8222543001174927, -0.27397462725639343, -0.01758819818496704], [0.6671196818351746, -1.4212297201156616, 0.784805178642273, -0.11470801383256912, 0.6883690357208252], [0.4704943895339966, -0.5563449859619141, -0.8128380179405212, -0.7319852113723755, 0.07098668813705444], [0.3830561339855194, 1.753658652305603, 0.8494983315467834, 1.1132537126541138, -0.2452573925256729], [1.34271240234375, 1.3709324598312378, -1.2741252183914185, -0.03894229978322983, 0.8531354069709778], [0.17814256250858307, -0.27840402722358704, 0.7415899038314819, -0.48934030532836914, 0.19071871042251587], [1.821691632270813, -0.4494190216064453, 0.1786351501941681, -1.9977781772613525, 0.7125278115272522], [1.9837530851364136, -0.38311755657196045, -1.804628610610962, -0.2125563621520996, -1.6439738273620605], [0.372898668050766, 1.2346107959747314, 0.8407620787620544, -1.9741735458374023, 2.269620895385742]], [[-1.8872296810150146, 2.2490508556365967, -2.3927407264709473, -0.0685480609536171, -0.9472435712814331], [0.016013918444514275, 0.412514328956604, 1.4746742248535156, 1.0367205142974854, -1.7262197732925415], [0.458685964345932, -0.26671093702316284, 0.3884653151035309, 0.35453712940216064, -0.11585266143083572], [-0.8408119678497314, 0.5803640484809875, 1.935826301574707, 0.1666433960199356, -0.3676814138889313], [-0.6362338066101074, 1.2780996561050415, -0.8852354288101196, 1.8303314447402954, 0.17804011702537537], [-2.0668604373931885, 1.1335030794143677, -0.8185120820999146, -1.3609956502914429, -1.6148028373718262], [-0.8925716280937195, 0.1363976001739502, -0.041629306972026825, 0.9385299682617188, 0.11465196311473846], [-1.1424397230148315, -1.9755349159240723, 0.7632198929786682, 1.7368512153625488, 1.5461575984954834], [-1.0612214803695679, -0.23758378624916077, 1.9129836559295654, 0.39654794335365295, -1.097297191619873], [0.19036641716957092, 0.29260456562042236, 1.1051254272460938, -0.8356581926345825, 0.27297064661979675]]], [[[0.6082843542098999, 0.4481736719608307, -0.5000821948051453, -0.6818879246711731, 1.1973731517791748], [1.2906732559204102, 0.8209097385406494, 0.8980866074562073, -1.2387152910232544, 1.412272334098816], [-0.601768970489502, -0.44018569588661194, -0.8141152262687683, -1.8893481492996216, 0.8354376554489136], [0.205281600356102, -0.9354938268661499, 0.4435957372188568, 1.2571161985397339, -0.4373035728931427], [-0.21895021200180054, 2.792917251586914, 0.09008839726448059, 0.7068283557891846, -0.874869704246521], [0.8755368590354919, 0.41181039810180664, 2.376220226287842, -0.7940782308578491, -2.0048208236694336], [-0.15821488201618195, 1.454566240310669, -1.6604501008987427, -0.018274910748004913, 0.7698227167129517], [-0.28448453545570374, 0.04156943038105965, 0.10168658941984177, 0.38306185603141785, 1.2710591554641724], [-0.4765428602695465, 0.14382438361644745, -2.021972417831421, -0.3983151316642761, -0.3167426288127899], [-0.933758020401001, 1.3952914476394653, 0.6352418065071106, 0.08079241216182709, -0.2621016204357147]], [[1.5475643873214722, 0.6876397132873535, -1.3480664491653442, 0.8322606682777405, -0.3094295263290405], [0.29030588269233704, -1.750824213027954, 0.6169719099998474, 1.0465784072875977, -1.2813528776168823], [0.182408407330513, 0.6537931561470032, 0.6570174098014832, -0.5917274355888367, -1.8351609706878662], [-0.2355767786502838, -1.356040120124817, -1.1268818378448486, -0.1736147105693817, -0.5588804483413696], [0.1285974383354187, 2.0233545303344727, 1.3586080074310303, 0.8645979166030884, 0.8392256498336792], [0.4472369849681854, -1.7880430221557617, -1.3160597085952759, 0.5598303079605103, -0.5731616616249084], [0.05700382590293884, 0.8413258790969849, 0.43287211656570435, -1.161295771598816, 1.2820168733596802], [0.04659273847937584, -1.4127001762390137, -1.4433835744857788, -0.9020309448242188, 1.1348381042480469], [0.28637760877609253, 0.08188751339912415, 0.4322303235530853, 0.9197400808334351, -0.29576125741004944], [-0.6327240467071533, 1.0320489406585693, -0.307350218296051, 0.16092908382415771, 1.1447004079818726]]]], "dtype": "float32"}, {"__nd__": [10, 4], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[-0.8358948826789856, -0.7492488026618958, 0.12689907848834991, 1.1851311922073364, 0.6127730011940002, 1.0300109386444092, -0.2475164383649826, -1.4746100902557373, -0.3076491951942444, -1.3540141582489014, -0.01741357147693634, 0.7104607224464417, 0.743986964225769, 1.2159383296966553, -0.37574219703674316, 1.7730376720428467, -0.6298396587371826, -0.8099997043609619, 1.3114712238311768, -0.16689112782478333, 0.759976863861084, -0.17460942268371582, -1.0795549154281616, -0.36925777792930603, -0.5502085089683533, -1.2503336668014526, -0.07240678369998932, 0.2531836926937103, 0.8721968531608582, 0.031627412885427475, 0.2870963513851166, -0.11965277791023254, -1.407834768295288, 0.3393809199333191, -0.6044778823852539, -0.3752758204936981, 0.09365470707416534, -0.05480523034930229, 0.24419109523296356, 0.47811460494995117, -0.4022423028945923, -0.22132913768291473, -0.8609272241592407, -1.1454435586929321, 0.8583798408508301, -1.2696712017059326, -0.5515924096107483, -2.001511573791504, -0.3503507077693939, 0.5629974007606506, 0.828862726688385, -0.6430014967918396, 0.6845542192459106, 0.474409282207489, 1.6913611888885498, -0.21907733380794525, 0.08807986229658127, -1.074515461921692, -0.07362519204616547, 0.6619321703910828, 0.19219522178173065, 0.7998368740081787, 0.8641991019248962, 1.1674867868423462, -0.31378549337387085, 0.8354954719543457, -0.1051129475235939, 1.1606414318084717, -0.23459872603416443, 1.1590453386306763, 0.37426188588142395, 0.6766332983970642, -0.2140592634677887, 0.37270545959472656, 1.3318008184432983, 0.067112497985363, -2.1730778217315674, -0.550142765045166, 1.0972775220870972, -0.796725332736969, 0.192660391330719, 1.399979591369629, -0.5523452162742615, -0.13846564292907715, -0.2695726156234741, -0.3539893627166748, -0.862855851650238, 1.0139062404632568, 0.062389109283685684, -1.1985834836959839, 0.3213369846343994, -1.8788800239562988, 0.4877175986766815, -0.7066534161567688, -0.6254885792732239, -0.5174050331115723, -1.1453018188476562, -0.010731996968388557, -0.165344700217247, -0.6736624240875244, 0.8495471477508545, -0.886945903301239, 0.2338714450597763, -0.6815134286880493, -0.11415020376443863, -1.5705606937408447, 1.361753225326538, 0.12384113669395447, -0.41812261939048767, 0.016547173261642456, -0.8600168824195862, -0.8474650382995605, 0.1360420435667038, -0.0674365684390068, -1.1020166873931885, 0.9173263907432556, 0.02365153469145298, 0.021719204261898994, 0.29907476902008057, -0.33612555265426636, 0.2767537236213684, -1.1601570844650269, -1.2136235237121582, -1.0432636737823486, -0.3259442448616028, 0.8724785447120667, -1.8923425674438477, 0.8694883584976196], [0.475953072309494, 0.7543489933013916, -0.2656908631324768, 1.09140145778656, -0.21005778014659882, -0.001925238175317645, -0.8911086320877075, 0.9520604610443115, -1.5674688816070557, -1.1466108560562134, -0.9196233153343201, -0.4729580283164978, -0.9733282923698425, 0.23120331764221191, 0.029361408203840256, -1.3260077238082886, 1.064221978187561, -1.227325677871704, 1.3823225498199463, 0.426570326089859, 1.73186194896698, 0.11092525720596313, -0.05604387819766998, 0.2460622936487198, -0.9465541243553162, -0.6916224956512451, 0.12102403491735458, 0.0013792855897918344, 0.19772763550281525, -0.7457913160324097, 0.8810874223709106, -0.4906722903251648, -0.31989389657974243, 0.9924588799476624, 0.3268784284591675, 0.5251232385635376, 0.1655815839767456, 1.3261557817459106, -1.848161220550537, 0.4298744797706604, 0.2372571974992752, 0.4716958701610565, 1.2857685089111328, 0.37814027070999146, 0.7550340890884399, -0.22688992321491241, -0.7702657580375671, 0.3617601692676544, 0.6430854201316833, 0.6583463549613953, 1.231358528137207, 0.5129940509796143, -0.33848312497138977, -0.09275828301906586, -0.8830127120018005, 0.3729389011859894, -0.11771812289953232, -0.6894017457962036, -1.236525297164917, -0.9231254458427429, -0.13837873935699463, 0.07822265475988388, -0.9768457412719727, 0.08161275088787079, -0.17861466109752655, -1.553594708442688, -1.029341459274292, 1.207220196723938, 0.7264966368675232, -1.4010790586471558, 0.15713781118392944, -1.7605313062667847, 0.08741699904203415, -0.6859370470046997, 0.3659684956073761, 0.0728130117058754, 0.34829583764076233, 1.4574073553085327, 1.436722755432129, -1.8146132230758667, 0.5226561427116394, -0.8185413479804993, 0.012819930911064148, 0.7474116683006287, 0.08139919489622116, 0.15234485268592834, 1.4225889444351196, -0.9996383786201477, 0.6400807499885559, -0.042412951588630676, 0.9086902141571045, 3.0921316146850586, 0.7839266061782837, -0.6227477788925171, 0.17778541147708893, 0.32571348547935486, -0.9284422993659973, -0.39678850769996643, -1.227599024772644, -0.5220226645469666, 0.29403096437454224, -0.002542076399549842, 0.3906453847885132, 0.12325960397720337, -1.0832880735397339, 0.8353636264801025, -0.11951188743114471, 0.22909128665924072, 0.09944914281368256, -0.06456764042377472, 0.21610553562641144, 0.2895278334617615, -0.7339441180229187, 0.5580937266349792, -1.9439536333084106, -1.2451632022857666, -1.1867036819458008, -0.45725348591804504, 0.7424998879432678, -0.4061613380908966, 0.4357427954673767, 1.5378718376159668, -0.42252635955810547, 0.11576105654239655, 0.2395445555448532, 0.6546375155448914, 1.8070108890533447, 1.3621262311935425], [1.0170321464538574, 1.0588847398757935, 0.1576298624277115, 0.31457382440567017, -0.05325412377715111, -0.2535417377948761, -0.07476823776960373, 1.501128077507019, -0.4325109124183655, 2.0552031993865967, -0.91200190782547, 1.009992003440857, -0.3071470260620117, -2.296644449234009, -0.09220189601182938, -2.664930582046509, 0.5276841521263123, -0.7500361204147339, 1.8403046131134033, 0.8194742202758789, 1.1704113483428955, -0.915047824382782, -0.2876967787742615, -0.2610924243927002, 1.1965878009796143, -1.7552076578140259, 1.0831156969070435, -1.0702850818634033, 1.0192794799804688, -0.6227821111679077, 0.5544255375862122, -0.011243721470236778, -0.11010430753231049, 0.021101539954543114, -2.2308976650238037, 0.4533381164073944, -0.4465513527393341, 1.5376086235046387, -1.9656249284744263, 0.0851474180817604, -0.32152268290519714, 0.32729578018188477, 0.24739699065685272, 0.1621924489736557, 1.4150608777999878, -0.5473925471305847, 0.4818195104598999, 0.646834671497345, 0.24268026649951935, 0.338950514793396, 0.23008129000663757, -0.5670547485351562, -0.6546372771263123, -0.08235310018062592, 1.15676748752594, 0.025444742292165756, -0.33531203866004944, -1.551836371421814, -0.29619473218917847, -1.044950246810913, -0.9614076018333435, -1.1025323867797852, 0.07862978428602219, -1.1084822416305542, -1.5446137189865112, -0.5847607254981995, -0.6442530155181885, 0.6879547238349915, -0.08193289488554001, -0.5971438884735107, -1.4553658962249756, 0.5663564205169678, 2.6495141983032227, 0.20371940732002258, 0.05823130905628204, -0.15510576963424683, 1.5492053031921387, 0.803160548210144, 0.7869621515274048, -0.2982945740222931, -0.9721071720123291, -1.0223579406738281, -0.8558235168457031, -0.1408332735300064, -0.9858003258705139, 1.8677364587783813, 0.37501809000968933, 0.24249683320522308, 1.0055216550827026, 1.84867525100708, 1.022654414176941, 0.2883763015270233, -0.6203710436820984, -0.6212392449378967, -0.5177543759346008, 1.9103879928588867, 2.6343741416931152, 1.9302927255630493, 0.1788681596517563, 0.4414910674095154, 0.10566366463899612, 0.18255673348903656, 0.33930426836013794, 1.31462824344635, -0.03725894168019295, -0.5650988817214966, -0.2854514718055725, -0.6545971035957336, -1.3163471221923828, -0.630376935005188, -0.1667557805776596, -0.637470006942749, 0.36744335293769836, 1.3628727197647095, 0.7857458591461182, 1.8764839172363281, 1.9221380949020386, 1.071738600730896, 0.015195643529295921, -0.7534221410751343, 0.8261788487434387, -1.1575794219970703, 1.2828010320663452, 1.3683311939239502, -0.8397308588027954, -1.0934491157531738, -0.580411970615387, 0.6597511172294617], [2.317634344100952, 0.4641222059726715, 0.9332907199859619, -0.5360661149024963, -1.0702284574508667, 0.11500363796949387, 1.5246080160140991, -0.5622966289520264, 0.8424624800682068, -0.6624135971069336, -1.0593228340148926, 0.44524458050727844, -0.3431186079978943, -0.30487391352653503, -0.18611164391040802, 0.5328001379966736, 1.4353892803192139, -0.537550687789917, 0.2853083312511444, -0.35247907042503357, -0.8018783926963806, 0.63531494140625, -0.23842988908290863, 1.0919742584228516, -0.07244020700454712, -1.3542460203170776, 2.7090566158294678, 1.0176312923431396, -1.2833925485610962, 0.6442765593528748, 0.10303002595901489, 1.031751036643982, -0.512749969959259, -0.5006945729255676, -2.161614179611206, 0.6657407283782959, 0.6238330006599426, 0.41479358077049255, 0.41491928696632385, -0.7902692556381226, -0.4910655915737152, 0.5120121836662292, -0.9943680763244629, 0.15936970710754395, -0.11538366973400116, 0.13079099357128143, 0.5035451054573059, -0.13996610045433044, 1.443910837173462, -0.45123764872550964, -0.37701767683029175, -0.7570580244064331, 0.9585309624671936, 1.220083236694336, -1.5901026725769043, 1.1516510248184204, -1.4304732084274292, -0.47351136803627014, -0.24418345093727112, 0.791208803653717, -0.21931475400924683, -1.4088791608810425, 0.9834260940551758, -0.3443974554538727, -1.3010470867156982, -0.3052740693092346, 0.9544335603713989, -0.6038057208061218, 1.0449633598327637, -0.20446766912937164, 0.09590154886245728, -0.7396836876869202, 0.30066820979118347, -0.5951815247535706, 0.20383289456367493, -0.46082744002342224, -0.2779134213924408, 0.8948066234588623, 0.6435213685035706, 0.5052830576896667, 0.5961781144142151, -0.4084109663963318, 0.18045422434806824, -1.180620789527893, -1.4858564138412476, -0.11434130370616913, -1.1242398023605347, -0.174966961145401, -1.6320512294769287, 0.7659502029418945, 1.5470277070999146, -0.1309356987476349, -0.07261138409376144, 0.8913955688476562, -0.31845009326934814, 0.39228391647338867, -0.9550431966781616, 0.4827600121498108, -1.1792380809783936, -1.5468748807907104, 0.4015391767024994, 0.10242252796888351, 0.07902009040117264, -1.294763207435608, 1.574658751487732, 0.14065244793891907, -1.0498907566070557, -0.7985584735870361, 0.5035898089408875, -0.4505000412464142, -1.7926627397537231, 0.950519859790802, -0.3577258288860321, -0.4728473126888275, -0.6637775301933289, 0.673334002494812, 0.7693292498588562, -0.10257050395011902, 1.3591543436050415, -0.5894454121589661, 0.05225636810064316, -0.4220919609069824, 1.7713656425476074, 0.21103298664093018, 0.49404940009117126, 0.9447611570358276, -0.7879204154014587, 2.170342206954956], [-1.0148751735687256, 0.5727707743644714, 1.5644240379333496, 1.995041012763977, -0.41768041253089905, -0.734237790107727, -0.6324328780174255, -1.7894244194030762, -1.7096079587936401, 1.5937193632125854, -0.6309462189674377, 0.762289822101593, -1.9577338695526123, 0.02588329091668129, -0.6351637840270996, 0.7345538139343262, -0.4725660979747772, 0.4792718291282654, 2.103844404220581, 1.496184229850769, 0.7962386012077332, 0.7679896950721741, -0.20135480165481567, -1.2836992740631104, 0.882170557975769, -0.5300740003585815, -1.0026642084121704, 1.381676197052002, -0.18006379902362823, 0.4454599618911743, 0.7281136512756348, -1.3658428192138672, 1.6920766830444336, -0.30974432826042175, -0.7498294115066528, 1.0466532707214355, 1.463366985321045, -2.20920467376709, 0.07086391001939774, -0.5761843919754028, 1.876914143562317, 1.136135458946228, 0.7930099368095398, -1.729326605796814, -1.942081093788147, 1.1236237287521362, 1.317883014678955, 0.17524465918540955, -0.40370848774909973, 0.2752409279346466, -0.7176842093467712, -1.0016788244247437, 1.0890885591506958, 0.42833226919174194, 0.7157416939735413, 0.5090904831886292, 0.854495644569397, 0.24677976965904236, -0.21485556662082672, -2.340585947036743, 1.1770886182785034, -0.18818990886211395, -0.855545699596405, 0.972065806388855, 0.6308205723762512, 0.8921454548835754, -0.33571314811706543, 0.2963554263114929, -1.4297205209732056, -0.020335406064987183, 0.07883059978485107, 0.6987726092338562, -0.4364575445652008, 0.2342159003019333, -0.9097254276275635, -1.6085044145584106, -1.6873626708984375, 0.34892404079437256, -1.3627227544784546, -0.037638407200574875, 0.14211463928222656, 0.26392078399658203, -0.9471501111984253, -0.415115624666214, 0.22416965663433075, -0.4768959581851959, -0.07123823463916779, -1.8777039051055908, 2.5706005096435547, -0.02723683789372444, 0.9223883152008057, 1.7292853593826294, 3.0042386054992676, -0.8349770307540894, -0.7297531962394714, -0.3158067464828491, -0.810405433177948, -0.9677082300186157, 0.7128776907920837, 0.03276598080992699, 0.5983373522758484, 0.9802088737487793, -0.005059042014181614, 0.30067679286003113, 0.4336963891983032, 0.06146116182208061, 0.6816322803497314, 0.18933601677417755, -1.594251275062561, 1.8516802787780762, -0.7723247408866882, 0.2760610282421112, 0.4683857262134552, -0.520218014717102, -0.12261997163295746, -0.6702357530593872, 0.10955989360809326, 0.12409321218729019, -0.8868231177330017, 0.5251514315605164, 0.059859029948711395, 0.5234819054603577, 1.071797490119934, 0.431262344121933, -0.43348610401153564, 0.13297565281391144, 0.2226899117231369, 1.4318784475326538], [-1.6578388214111328, 0.29400110244750977, 0.0339222215116024, -0.8935646414756775, 0.893549919128418, 0.31426891684532166, -1.5266846418380737, 0.097648985683918, 2.5166265964508057, -2.1129019260406494, 0.39228716492652893, -0.2816097140312195, 1.5257277488708496, -0.47934406995773315, 0.032824426889419556, 0.14955087006092072, 0.7729616165161133, -0.38365814089775085, -0.08329722285270691, 1.364730715751648, -1.0578181743621826, -0.6077969670295715, -0.6904796957969666, -0.37923958897590637, 0.701796293258667, 0.7651252150535583, 0.9863858819007874, 0.7311770915985107, -1.2199931144714355, -0.5524089336395264, -0.6064444184303284, -0.7909141182899475, -0.12936373054981232, 1.8565397262573242, 0.39214202761650085, -0.3200754225254059, 0.9244135618209839, -1.0805565118789673, -0.4779898226261139, -0.616401731967926, -0.5337314009666443, -0.20328645408153534, 1.3433773517608643, -0.3232789635658264, -0.13890333473682404, -1.4337339401245117, -1.0357264280319214, 1.0068706274032593, 0.42320600152015686, -0.7944551110267639, 0.02751559391617775, 0.7661355137825012, -1.585702657699585, 0.23805958032608032, 1.0970360040664673, 1.532659888267517, 1.1489118337631226, -0.6120585799217224, -0.6388119459152222, 0.11400896310806274, -1.455405592918396, -0.8001040816307068, 0.4514569640159607, -1.2543673515319824, 1.549737572669983, 0.5399158596992493, 1.3486632108688354, -0.33256298303604126, -0.9248870611190796, 0.2773328423500061, -1.8104209899902344, -0.2766287922859192, 0.8609151840209961, -0.20586316287517548, -0.519418478012085, 0.21517786383628845, 0.9052873849868774, -1.4424010515213013, 2.438474178314209, 0.545536994934082, 0.0576489083468914, 0.21373111009597778, 0.9957472085952759, 0.7587278485298157, -0.6985112428665161, -0.9697425961494446, -0.4575822949409485, 0.05077027529478073, 0.7379735112190247, 0.13245674967765808, 1.2510640621185303, 0.5004023313522339, -0.2330016940832138, -0.07991482317447662, -0.6538897156715393, -0.8810664415359497, -0.6469339728355408, -1.1413915157318115, -0.6272270679473877, -0.34584030508995056, 1.1056169271469116, -0.026563094928860664, 1.852360725402832, 2.0832159519195557, 0.15319998562335968, -1.4447638988494873, 0.4346430003643036, 0.12737280130386353, -0.16484132409095764, 0.22960589826107025, -0.6053653955459595, 0.018273279070854187, 0.6618918776512146, 1.3784006834030151, -0.2590155303478241, -0.11556745320558548, 1.3950185775756836, -0.6639583706855774, -1.0620027780532837, 0.256256639957428, -0.5868607759475708, -0.610596776008606, -1.8399771451950073, -0.19949372112751007, 0.9648222327232361, 1.4648369550704956, 0.45386284589767456, -0.2558766305446625], [0.377704381942749, -0.2380003184080124, 0.2873415946960449, -1.1908379793167114, 0.9592229723930359, -0.046990517526865005, -1.028624176979065, 0.45729508996009827, 1.3457969427108765, 0.32127705216407776, 0.4814501702785492, 0.2634153664112091, 0.6671515703201294, 0.6821397542953491, 0.19687587022781372, 0.8088603615760803, 0.04119270667433739, -0.6221162676811218, -1.339422345161438, 0.7266095280647278, -0.7369012236595154, -0.833522617816925, 0.3589756488800049, 1.4004418849945068, 1.0090843439102173, 0.48459887504577637, 1.268250584602356, -1.908002257347107, -0.526794970035553, 1.2722041606903076, -0.17821629345417023, 0.8138281106948853, -0.25294092297554016, -1.0799450874328613, 0.4599095582962036, -0.681571900844574, -0.5159640312194824, 0.8902517557144165, -0.20690129697322845, -1.4892030954360962, 0.727898895740509, 1.3066072463989258, -1.629830241203308, 0.7036844491958618, -0.7722615599632263, -0.8739343881607056, 2.194603443145752, 0.4359895586967468, 0.739591121673584, -0.49979451298713684, -1.1964937448501587, 0.13110433518886566, -0.5122163891792297, -0.31255486607551575, 0.7013121843338013, -0.352039098739624, -0.7527945637702942, 2.1542398929595947, -0.37460216879844666, 0.6330841183662415, -0.46503421664237976, 1.7002875804901123, -0.9740995168685913, 0.628663182258606, -0.06693953275680542, 0.07772330939769745, -0.9790493249893188, 0.2913842797279358, -0.42835837602615356, 0.7645682096481323, -0.6208759546279907, -1.0353978872299194, -0.18148095905780792, 1.1180344820022583, -0.049602195620536804, -0.7008435726165771, -0.6257852911949158, -0.6792227029800415, 1.728961706161499, 0.7088293433189392, 0.3301137685775757, -0.9619460701942444, -0.5938128232955933, 0.7794328331947327, -1.2300145626068115, 0.6742097735404968, -0.383110374212265, 0.19750681519508362, 0.6440991163253784, -0.22438964247703552, -0.8863945007324219, 1.9284522533416748, 1.6695408821105957, -2.99130916595459, -0.2283172756433487, -0.5240658521652222, 1.1552497148513794, -1.1833778619766235, 0.5871095061302185, 0.7604048848152161, 0.453437864780426, 0.27427998185157776, -0.25279560685157776, 1.8582361936569214, 0.3568568527698517, 0.5719924569129944, -0.9958747029304504, 0.5415090322494507, 0.9722817540168762, -0.13116605579853058, -0.8673829436302185, 0.7029394507408142, -0.15922020375728607, -0.38762351870536804, -1.1842714548110962, 0.282979816198349, -0.5516008734703064, 1.320663332939148, 0.19337205588817596, -0.5382242798805237, -0.05696496739983559, -0.5780189633369446, -0.3622652590274811, 0.7867183089256287, 0.007981694303452969, -0.050870634615421295, 0.5986065864562988, 1.3759996891021729], [-0.03144845739006996, -0.3479301929473877, -1.0073498487472534, 0.2890443801879883, 0.3540019392967224, -0.08893460035324097, -1.0436091423034668, -0.5608684420585632, 1.1219911575317383, -1.6510590314865112, 1.1139763593673706, 0.5492724180221558, -0.7973261475563049, 0.6113355159759521, 2.1136956214904785, 1.3068090677261353, 0.6670776605606079, -1.0916715860366821, 1.8940644264221191, 0.15504568815231323, -2.6306841373443604, 0.6888604760169983, 0.24163898825645447, -0.06794705986976624, -0.38706716895103455, -0.001329214544966817, -1.1946921348571777, -0.03842289745807648, -0.7059695720672607, 0.983269453048706, -1.9312294721603394, 1.1229820251464844, 1.3213541507720947, 1.8286609649658203, 0.546682596206665, 2.1593523025512695, 1.5258740186691284, -0.4537758529186249, 0.3329174220561981, -1.4556770324707031, -1.7724369764328003, 0.4028996229171753, 0.3836289048194885, -0.8109895586967468, -1.153381109237671, -0.869113564491272, 0.39523056149482727, 0.35115525126457214, 0.12550264596939087, -0.7370885014533997, 0.9446954727172852, 1.354262113571167, -1.2691246271133423, 0.41972988843917847, 0.2748650014400482, -0.41738012433052063, 0.062212925404310226, -0.5591364502906799, -0.5670019388198853, -0.17055334150791168, -0.5414919853210449, -0.6436651945114136, -0.9240917563438416, -0.3895909786224365, 0.027925703674554825, -1.2624238729476929, 0.6948047876358032, -1.4214355945587158, 1.6696504354476929, -1.105216383934021, -2.6395578384399414, -0.8920459747314453, -0.8474315404891968, -0.05741894990205765, 0.38918450474739075, 1.290188193321228, -0.28656068444252014, -0.6804601550102234, 1.910409927368164, 0.7121158242225647, -0.19764645397663116, 0.21351002156734467, -0.5051677823066711, 0.7244899272918701, 0.23529528081417084, -0.9729953408241272, -1.2811263799667358, 0.7115157246589661, -0.04221860319375992, -0.4167034924030304, -1.0354317426681519, -0.28475499153137207, 0.4809953570365906, -1.703911542892456, 2.4613072872161865, -0.8967798352241516, 1.8159680366516113, 0.10917644947767258, 1.5352399349212646, -0.2865552604198456, -1.158981204032898, -0.2943342626094818, -0.41131168603897095, 0.09129966050386429, 1.0535980463027954, -1.0243494510650635, 0.5897109508514404, 0.9499473571777344, 0.19140397012233734, -0.6202195286750793, 1.286179542541504, 1.198706030845642, 0.9827528595924377, 0.25231993198394775, 1.6330840587615967, -1.313454031944275, 0.6936138868331909, 1.5228016376495361, 1.2578656673431396, -1.431409478187561, 0.7361136674880981, 1.8885295391082764, -0.4843498170375824, -1.698805332183838, -0.9683853387832642, 0.5747280120849609, 1.3193913698196411, -1.2428131103515625], [0.95982426404953, -0.6710734367370605, -1.226505160331726, -0.3398974537849426, -0.21387100219726562, -0.3578338921070099, 1.3507121801376343, 2.0803439617156982, -0.13370034098625183, -0.2339482605457306, -1.749214768409729, 0.7462241649627686, -0.5563958883285522, -0.3117798864841461, 0.8527589440345764, -1.3016209602355957, -0.1770579069852829, -1.9441509246826172, 0.28382429480552673, 0.7264275550842285, -0.27921757102012634, -0.27317413687705994, -1.0340163707733154, -1.0922547578811646, 2.2001311779022217, -1.467574119567871, -0.42971083521842957, -1.0709656476974487, -1.9145039319992065, 0.5608466267585754, -1.138522744178772, 1.2942264080047607, 0.6678308844566345, 1.492601752281189, 0.10418812185525894, -0.08671346306800842, -0.6209465861320496, 0.5910029411315918, -0.6406233906745911, 1.6605756282806396, -0.9725055694580078, 0.6361387372016907, -0.03333841264247894, -0.26799967885017395, -0.32760220766067505, 1.038413643836975, 0.8235840201377869, -0.02223372459411621, -0.06741315126419067, -0.7484764456748962, -0.7365787625312805, -0.09406916052103043, -0.5889838337898254, -1.5563621520996094, -0.3202938437461853, 1.6770884990692139, 0.751896321773529, -0.06607294082641602, 1.0512661933898926, -0.5021456480026245, 0.3442901074886322, -0.49515217542648315, -1.5327099561691284, -0.1248701885342598, 0.3986944556236267, 1.3062658309936523, -0.404079407453537, 0.6713934540748596, -0.49285387992858887, 0.8091824650764465, -0.4801160991191864, 1.1916245222091675, -0.5335688591003418, 0.5650676488876343, -0.49573567509651184, -0.8254448175430298, -0.9492902755737305, -1.0713891983032227, 1.6466501951217651, -0.5980514883995056, 0.2677907347679138, 1.1912490129470825, -1.2765206098556519, 0.09571308642625809, 0.05789751932024956, -0.08764170110225677, 0.1090737134218216, 0.5822839736938477, 1.1477991342544556, -0.4403032660484314, -0.08330415189266205, 0.6194721460342407, -0.32200273871421814, -0.5299765467643738, 0.9632738828659058, 1.219870924949646, 0.20568180084228516, 0.30141404271125793, -0.06572018563747406, 0.7685028910636902, -0.6470226049423218, -0.3129523694515228, 0.7508363127708435, 0.025832049548625946, -1.4275296926498413, 0.4752689003944397, 1.1094118356704712, 0.23114220798015594, 0.4294479191303253, 0.7382975816726685, -0.5149836540222168, 0.8096388578414917, -0.3510623872280121, -1.0460355281829834, -0.7116126418113708, -0.11534764617681503, 0.36828693747520447, 0.8948222398757935, -1.630694031715393, -0.7024412155151367, -0.8779889941215515, 1.8513773679733276, -0.2612711787223816, 0.2793801426887512, 0.7146572470664978, 0.3177106976509094, 1.13005793094635, -1.1528215408325195], [0.7703520059585571, 1.3324954509735107, 0.9360408782958984, 0.41417816281318665, -1.7768007516860962, 1.5103065967559814, 1.5747824907302856, 0.3676993250846863, -1.885646939277649, -0.05939231812953949, -1.344782829284668, -0.032565124332904816, -0.17391858994960785, 1.2989064455032349, -1.2855966091156006, 0.14010469615459442, -0.8834280967712402, -0.7489727139472961, 0.28396138548851013, 2.0690367221832275, -0.30225709080696106, -2.028686046600342, 1.3782014846801758, -0.13622251152992249, -1.1665276288986206, -0.6686456203460693, -1.0978144407272339, 1.2967798709869385, 0.20340681076049805, 0.39973023533821106, 1.3061437606811523, 0.9924890995025635, 1.3622742891311646, -0.7001648545265198, -0.5736575722694397, -0.8392177224159241, -0.30907899141311646, 0.05053792521357536, -2.5709218978881836, 0.8998321890830994, 0.06359683722257614, 0.5740432739257812, -0.19338852167129517, 0.9018384218215942, 0.6780436038970947, 1.2971750497817993, -1.8571518659591675, -0.3134533166885376, 0.10926979035139084, -0.7485412359237671, -0.041685860604047775, 0.4360748827457428, 0.9507409930229187, 0.08263125270605087, -1.5193216800689697, 0.024291792884469032, -0.7571995854377747, -0.1973535716533661, 0.40623563528060913, 0.4869012236595154, 0.3206127882003784, 0.9890263080596924, -0.0042726825922727585, 0.9661753177642822, -1.625704050064087, 2.329393148422241, 1.5019418001174927, 0.9289839267730713, -1.9290554523468018, 0.8765773177146912, 0.030877448618412018, 0.7006205916404724, -1.1638448238372803, -1.0311291217803955, 0.6532650589942932, -0.5051576495170593, -1.165086030960083, 0.29063379764556885, -0.46513256430625916, -1.4899908304214478, 0.22223906219005585, -0.1401837021112442, 0.8350504040718079, -0.3750828504562378, 0.2734629809856415, -0.17630359530448914, -0.472491592168808, -0.04398516193032265, 0.674040675163269, 0.2818588614463806, -0.5247704982757568, -0.32427600026130676, -0.31513869762420654, -0.38651368021965027, 0.49218615889549255, -0.3475703299045563, -0.7179967164993286, 0.6995590925216675, 1.1179192066192627, -0.9174093008041382, 0.27545106410980225, -0.6332220435142517, 0.8225661516189575, -1.6808053255081177, -1.666079044342041, 1.0115206241607666, 0.9779115915298462, 0.980930745601654, -0.875335693359375, -0.19932761788368225, 0.407335489988327, -1.306712031364441, -0.5254254937171936, -1.0679564476013184, -0.7849779725074768, 0.7246192097663879, -2.331592559814453, -1.2546402215957642, 0.5330635905265808, 0.12917938828468323, -0.41358068585395813, -0.24865184724330902, 0.5803433656692505, 0.708893895149231, -0.41416412591934204, 0.300778865814209, -1.5648088455200195, -0.29681456089019775], [-1.7853127717971802, -0.15081332623958588, -0.4000321626663208, 0.049646053463220596, -0.6624380946159363, -0.9702860713005066, -1.1625686883926392, 0.23886623978614807, -0.7224103808403015, 0.31410738825798035, 0.9311237335205078, 1.6793855428695679, 2.167529582977295, 0.1512748748064041, 0.31447890400886536, -0.270940363407135, 0.4461929500102997, 0.9314620494842529, 0.3223262131214142, 0.3966730833053589, -0.7367970943450928, -0.2794441878795624, 1.3225395679473877, -2.5148329734802246, -0.07270157337188721, -0.5907094478607178, 0.5183554887771606, -0.4526333808898926, 0.05946575105190277, -0.7407355904579163, 0.025952471420168877, -0.08140141516923904, -0.47028255462646484, -0.36385494470596313, -1.6922768354415894, -0.26053372025489807, -1.017321228981018, -0.38105809688568115, -1.6402114629745483, 0.19181686639785767, -0.739232063293457, -0.17645694315433502, -1.355384111404419, -0.2514919936656952, 0.8333863615989685, -0.3275006413459778, -0.8390824794769287, -1.1766612529754639, 1.6503393650054932, 0.8562300801277161, 1.4443542957305908, 0.4559018015861511, 0.7283895015716553, -0.6010465621948242, 1.1884969472885132, -0.09783896803855896, 0.3237132728099823, -0.3091692328453064, 0.6899526715278625, 0.49248161911964417, -0.7248042821884155, -0.04111713543534279, 0.15855205059051514, 0.44541317224502563, -0.20097893476486206, 1.4427436590194702, 0.9577332139015198, -1.2005105018615723, 0.4760059118270874, -0.29688760638237, -0.8435278534889221, 0.9033945798873901, 0.7320290207862854, 1.3386262655258179, 0.32177233695983887, 1.2957810163497925, -0.02166183479130268, 0.9606038928031921, 0.5161052346229553, 0.7067704796791077, 0.6589914560317993, -0.3497709333896637, -0.9404901266098022, -1.027974009513855, -0.592355489730835, -1.8310673236846924, -0.5050987601280212, 1.6315999031066895, -0.4798761308193207, 1.8481477499008179, 0.3542715907096863, 0.667157769203186, 0.5273805260658264, 2.598446846008301, -1.2199510335922241, -0.1264101266860962, -0.06621861457824707, 0.9506892561912537, 2.2723617553710938, 0.8597820401191711, 0.6084009408950806, -0.3769833445549011, -0.6140115857124329, -0.6302647590637207, 0.4164595901966095, 0.9110977649688721, -0.4488503038883209, -0.6631534099578857, -0.15684349834918976, -0.9585694670677185, -0.7046802639961243, -1.1722886562347412, -1.9107766151428223, 1.3526983261108398, 1.2447832822799683, -0.17372524738311768, -1.1514196395874023, -0.641072690486908, -0.8582776188850403, 0.6328040361404419, -0.3100089430809021, -0.12106157094240189, 0.1411079615354538, 0.059718601405620575, -0.3042883276939392, -0.4676593542098999, 1.924280047416687, 0.5882667303085327], [0.22201497852802277, -0.825310230255127, 1.0493254661560059, 0.4837816059589386, 0.2771356999874115, 1.1302882432937622, 0.4466073215007782, 1.3730850219726562, 1.0600801706314087, -0.35941261053085327, 0.29871442914009094, 0.11735375970602036, -0.46270042657852173, 0.21547754108905792, 1.881414532661438, 0.5977687239646912, 0.22941482067108154, 1.2952392101287842, -1.3581748008728027, 0.05223029479384422, 1.2087210416793823, 0.786665678024292, -1.2966890335083008, 1.0583134889602661, -0.5034135580062866, -0.417125403881073, -1.8627322912216187, 1.4723920822143555, -0.05572802573442459, 1.3327666521072388, 1.1291345357894897, 1.402080774307251, 0.07086050510406494, 0.4109205901622772, 1.7463186979293823, -0.05157484486699104, 0.05795607715845108, 0.06178279221057892, 0.922940194606781, 0.5914769172668457, 2.0046579837799072, -0.4803936183452606, -0.029344480484724045, -1.089215636253357, -0.21820957958698273, 1.2439682483673096, -1.0281904935836792, 0.8043040633201599, -1.0205543041229248, -0.1639208197593689, -2.1437246799468994, -0.037690624594688416, -0.5866401195526123, 0.9661582708358765, 2.6291513442993164, -0.8946570158004761, 1.1091402769088745, 0.2503519654273987, -1.0028581619262695, -1.1750417947769165, -0.35496246814727783, 1.8394386768341064, -1.494105339050293, 1.3088834285736084, 0.7413603663444519, -1.2675155401229858, -0.8938004970550537, 0.027823535725474358, -1.6748294830322266, 0.8567380309104919, -1.1375222206115723, 0.9724658727645874, -1.9346505403518677, 0.9344436526298523, -0.502798318862915, 0.07816492021083832, 0.8372974395751953, -1.3022775650024414, -0.32449743151664734, -0.2747761905193329, 1.9095745086669922, 1.369592547416687, 0.06727497279644012, 0.31386882066726685, -0.6827303171157837, 3.1868093013763428, 0.48806527256965637, 0.23080043494701385, -2.104165554046631, -1.099239468574524, 0.8684420585632324, -0.3566846549510956, 0.41742822527885437, -0.6272463202476501, -0.023271147161722183, -1.925707459449768, -1.831377387046814, -0.8740531206130981, -0.009234629571437836, -1.2198766469955444, -3.2064292430877686, -0.007831145077943802, -0.7207728624343872, -1.4791213274002075, -1.7304842472076416, 0.49644073843955994, 1.4909183979034424, 0.2844180166721344, 1.5888993740081787, -1.5118601322174072, 0.30797621607780457, -1.5110819339752197, 0.4698239266872406, -0.5166385173797607, -1.7970994710922241, 1.041196584701538, -0.7534015774726868, 0.25100523233413696, -0.2050403654575348, 0.567503035068512, 0.05643298476934433, 0.6638681888580322, 1.3000216484069824, -1.7656530141830444, 0.2509862780570984, 0.9761643409729004, -0.056988675147295, -1.5491487979888916], [-0.46830376982688904, -0.26096558570861816, -0.10488025099039078, -1.2206637859344482, -0.009622384794056416, 0.49300307035446167, 0.05860695242881775, -1.0403059720993042, 1.8297832012176514, 1.3271164894104004, -2.016213893890381, -0.6377583742141724, 1.3280575275421143, -1.1335264444351196, 0.5406962037086487, 0.9593963027000427, -2.0555036067962646, 0.8223453760147095, 0.9657633900642395, -0.8715633153915405, -0.1449470967054367, -0.7353242635726929, -0.28206557035446167, -0.9480443596839905, 1.4154942035675049, -0.6959812045097351, -0.9025841355323792, -0.2192663550376892, -1.6755030155181885, -0.23684659600257874, -0.18454201519489288, -1.0751020908355713, 0.3577481806278229, -0.5504014492034912, -0.9288245439529419, 3.518751382827759, 0.4594692289829254, 0.22687789797782898, -0.18868482112884521, 1.1220850944519043, -1.5147383213043213, 1.6346534490585327, 0.15004830062389374, 0.7452393174171448, -0.6045106053352356, -0.7560360431671143, 1.197738766670227, 0.6030040383338928, -0.08708606660366058, 0.6175930500030518, -1.0120733976364136, 2.7930169105529785, 0.12339857220649719, 0.45719876885414124, -0.06835391372442245, -0.13776984810829163, 0.7231460213661194, 0.1330215036869049, 1.1748722791671753, -0.531264066696167, -0.6107739806175232, -0.6991296410560608, -0.13000397384166718, 0.7378265261650085, 1.3347147703170776, -0.49625423550605774, -0.16696158051490784, 1.1136778593063354, 1.6219419240951538, -0.43126246333122253, 0.040963511914014816, 0.9920684695243835, -1.9328572750091553, -0.41132625937461853, 0.7708724737167358, 0.015872841700911522, -1.0139471292495728, -0.9995001554489136, 1.1484726667404175, 0.8885493278503418, -0.09056151658296585, 1.2929697036743164, 0.27659356594085693, 1.1859238147735596, 0.964833676815033, 0.587320864200592, -1.2859059572219849, 0.030101845040917397, 1.2193812131881714, 0.12335680425167084, 0.0035535763017833233, 0.06500809639692307, -1.3914680480957031, -0.25070858001708984, 1.0160670280456543, -0.5895722508430481, -1.3754440546035767, -0.9886310696601868, 0.24861331284046173, 1.433512806892395, 1.3255211114883423, -1.5125471353530884, 1.3714091777801514, 1.7262159585952759, 0.7102624773979187, 0.5017229318618774, -1.120277762413025, -1.8966102600097656, -2.1004607677459717, 1.2690699100494385, -0.419701486825943, 2.2115068435668945, 1.2436772584915161, 0.4920459985733032, 1.715101718902588, 0.3748534023761749, -0.9492154717445374, -0.0963742807507515, -0.10158959031105042, -0.21408817172050476, 0.4978778660297394, -0.30592212080955505, 2.254148483276367, -0.02783900313079357, 0.5233485698699951, 0.33845528960227966, 2.0880579948425293, -0.8802734613418579], [-0.09685534238815308, -0.42455676198005676, -0.06631786376237869, 0.4653554856777191, 0.07762712240219116, 0.875800371170044, 0.9099562764167786, 1.8397501707077026, 0.13544268906116486, -1.1709829568862915, 0.6720257997512817, -1.1553623676300049, 1.2752114534378052, 0.5028891563415527, -0.5226072072982788, 0.26588839292526245, 0.5414416790008545, -1.908053994178772, 0.7178407907485962, 0.8176060914993286, 1.1672877073287964, -1.0831577777862549, 0.1908620297908783, 0.6445098519325256, -0.09629300981760025, 0.10035308450460434, -2.7999343872070312, -0.8576048612594604, 2.095841884613037, -0.37961193919181824, 0.5303339958190918, 0.7724264860153198, -1.789976716041565, -0.43304023146629333, 0.019092099741101265, -0.8704396486282349, -0.405965119600296, -0.8830041289329529, -0.03694120794534683, 0.20337386429309845, -0.15640291571617126, -1.0084367990493774, 0.7341527938842773, -0.5101417899131775, -0.9838986396789551, 0.5265333652496338, -1.2068284749984741, -0.6122155785560608, -0.19389398396015167, -0.092678964138031, 0.6380778551101685, 0.025107506662607193, -0.4135475158691406, -1.0107030868530273, 0.6980884075164795, -1.7849371433258057, -0.7097217440605164, 0.0020787238609045744, 0.6126374006271362, 0.07265441119670868, -1.631847858428955, -0.2776798605918884, -0.21648600697517395, -0.5280066728591919, 0.4089539051055908, 0.5135959982872009, -1.1121610403060913, -1.6571325063705444, -0.7844823598861694, -0.2070075422525406, -0.22883255779743195, -0.7988958954811096, -0.13902176916599274, 0.35767295956611633, 1.1424999237060547, -1.2609375715255737, 0.16098633408546448, 0.001285675447434187, 0.17974241077899933, 0.5583406686782837, 0.4944646954536438, 1.9739692211151123, -1.740859031677246, 0.16776426136493683, -1.9414232969284058, -0.2284032553434372, 0.3443758487701416, 0.5194586515426636, 0.3407600224018097, 0.7003224492073059, -0.7299378514289856, -0.6058750152587891, 1.398093342781067, -1.0266343355178833, 1.1197314262390137, -0.4151467978954315, 0.401166170835495, -0.6226645708084106, 0.1861235797405243, 0.5935116410255432, -0.5146067142486572, -1.1685324907302856, 0.01690761372447014, 0.8996232151985168, -0.21726611256599426, -0.09879586100578308, -1.6794990301132202, 0.35269972681999207, 0.5984217524528503, -0.7314910292625427, 0.8741938471794128, 2.183398485183716, -1.2591501474380493, -0.8338804841041565, 1.003108263015747, -0.3480583429336548, 2.023637294769287, -0.14441116154193878, 1.0303740501403809, -0.8925147652626038, -0.4724208116531372, 1.9606989622116089, -0.25419896841049194, 0.8153796195983887, -0.20321476459503174, -0.705878734588623, -1.8100229501724243, -0.5055094361305237], [0.714831531047821, -0.3173018991947174, -1.2941776514053345, -1.4639673233032227, -1.0854027271270752, 0.6021865010261536, -0.2567180395126343, -0.4536289572715759, -0.31264811754226685, -0.4917534291744232, -0.5720130205154419, 1.3101252317428589, -1.8691959381103516, 0.28621914982795715, 1.0683305263519287, -0.24423938989639282, -0.29459908604621887, 0.20619729161262512, -0.2657933235168457, -1.329913854598999, -0.4762304127216339, 1.0662142038345337, -1.0366237163543701, 1.6454713344573975, 0.8234358429908752, -0.46964120864868164, -0.8931422829627991, 1.7702890634536743, -0.21018445491790771, -0.7250217199325562, 0.3537501096725464, 0.3645409643650055, -0.305979460477829, -0.5063034296035767, -0.07839080691337585, 1.1358263492584229, -0.41713929176330566, -1.003495693206787, 0.45921552181243896, -0.32046425342559814, 0.5244039297103882, -1.4868748188018799, 0.11626992374658585, 1.3827509880065918, -0.3529502749443054, 0.3415294885635376, 0.05350082367658615, 1.0003392696380615, -0.029584065079689026, -0.3459717631340027, -0.7965437173843384, -1.7697259187698364, -1.3170711994171143, 1.0991003513336182, -0.591560959815979, 0.9867937564849854, -0.3804602324962616, 1.3834749460220337, 1.6948285102844238, 0.575731098651886, 0.859587550163269, -1.4052660465240479, -0.6975130438804626, 1.2662943601608276, 1.767187476158142, 1.728770136833191, 0.2546926736831665, 0.15920689702033997, 0.5027711987495422, 0.09844600409269333, 0.05021010339260101, 3.049191474914551, -1.2106220722198486, 0.19160282611846924, 1.6242462396621704, 0.13392896950244904, -0.31876906752586365, -0.665732741355896, -0.40941521525382996, 1.2953684329986572, -1.8049763441085815, 2.765833854675293, -0.5225287079811096, -1.577609896659851, 1.2620524168014526, 0.49012601375579834, 2.7688348293304443, -1.3849384784698486, 0.025090020149946213, -1.052754521369934, -1.8083442449569702, 0.25578877329826355, 0.11071525514125824, -1.7096682786941528, 0.4000265598297119, -1.037412405014038, -1.3169554471969604, -1.2384299039840698, -0.9096704125404358, -0.8472113013267517, 0.5423204898834229, 0.22807036340236664, -0.3610687255859375, -0.8458987474441528, 0.47804179787635803, 0.16292904317378998, 0.8911266326904297, 0.6079750657081604, -2.151214122772217, 1.2050827741622925, -0.41306307911872864, -1.4391167163848877, -1.4050074815750122, -2.9402096271514893, 0.24221239984035492, -0.4347976744174957, 0.3499106168746948, 0.04636004567146301, 0.6569357514381409, 2.0511252880096436, 2.5112674236297607, 1.8448987007141113, 0.44662418961524963, 1.9522531032562256, -0.617970883846283, 1.3505008220672607, -0.9971863031387329, 0.2233327478170395], [-0.5478591918945312, -1.9262688159942627, -0.8238412141799927, 0.13340675830841064, 0.1508740782737732, -0.22408032417297363, -0.044701989740133286, 0.26849234104156494, 0.8815132975578308, -0.14826415479183197, 1.27024507522583, -0.08308230340480804, -0.5193064212799072, 0.6252172589302063, -0.4810872972011566, 0.16924957931041718, 0.15029126405715942, 0.040379561483860016, 0.6857789158821106, -0.21452437341213226, 1.031296730041504, -1.3047051429748535, -0.07950279861688614, 2.663667917251587, -0.700077474117279, 0.8567817807197571, 0.3787931203842163, 0.030606135725975037, -0.6618329882621765, -1.7467548847198486, -0.19473101198673248, 1.284257173538208, 0.5719641447067261, -0.4866505265235901, -2.1547534465789795, -0.8815433979034424, -1.5028785467147827, -0.671709418296814, 0.5039470195770264, 1.0365077257156372, -0.14219510555267334, -0.4622913599014282, 1.236851453781128, 0.4361576735973358, 0.384895384311676, -0.5007978677749634, -0.17428675293922424, 1.5182944536209106, -0.5330233573913574, -0.26464107632637024, -2.278564214706421, 0.78206467628479, 0.5159997344017029, 1.7313168048858643, 0.05596008151769638, -0.13113009929656982, 1.2935315370559692, 0.09583813697099686, 0.3917151987552643, -0.7229489088058472, -0.17037959396839142, 0.18915262818336487, 2.319948196411133, 0.5499929785728455, -1.0879515409469604, 0.22610539197921753, -0.6433438658714294, 0.8096803426742554, 0.48140373826026917, -0.03988303616642952, 1.100003957748413, 0.8616591095924377, 1.2388311624526978, -0.5470274686813354, 0.37980592250823975, 1.3019782304763794, -1.422545313835144, -0.8051163554191589, 1.6887257099151611, 0.2684843838214874, -0.040050532668828964, 1.0474363565444946, 0.5620158314704895, -0.4296327233314514, 0.6709493398666382, -1.931365966796875, -1.4202057123184204, 1.0117751359939575, -0.5158944725990295, 0.6588932275772095, -1.062092900276184, -0.4752914309501648, 0.10896942764520645, -0.1622089296579361, -0.4149434566497803, 0.013868732377886772, 1.8453311920166016, 0.5282554626464844, -0.7254921793937683, 0.7150225043296814, -1.1059908866882324, 1.449720859527588, -1.1551854610443115, -1.3830605745315552, -0.7547379732131958, -0.035617150366306305, -0.07616342604160309, 1.1730700731277466, 0.15202774107456207, -0.05107024312019348, -0.8404272794723511, -0.009613746777176857, -0.79011470079422, -1.4587125778198242, 1.376145839691162, 1.1739821434020996, -0.013660840690135956, 1.2414034605026245, 0.7518717646598816, 0.2850859463214874, 1.2567987442016602, -0.8305980563163757, 0.33954688906669617, -0.6330119371414185, 0.007817374542355537, -1.130264163017273, -0.8382492065429688, 0.08776146918535233], [-0.6402801275253296, -1.4363415241241455, -1.7293576002120972, -0.540558934211731, -0.9613327383995056, -0.6453918218612671, 0.16350992023944855, 0.9301466345787048, 0.33634206652641296, -0.34599733352661133, 0.0665067508816719, -1.1745121479034424, -0.5947383046150208, 0.11392145603895187, -0.19709721207618713, -0.4981481432914734, 0.8895739912986755, 0.40673720836639404, 0.8836843967437744, -0.8154907822608948, 0.4991302788257599, 0.6931819319725037, 1.4562242031097412, -0.3765331208705902, 0.5969178080558777, 0.98912513256073, 0.00554424524307251, -0.27162349224090576, 1.3366196155548096, 0.32621607184410095, 0.21104952692985535, 0.15316589176654816, -0.46222618222236633, 1.2332342863082886, -0.05183086916804314, -0.22485943138599396, 0.831416130065918, 1.9108394384384155, -0.8085787296295166, -1.5597418546676636, 1.2411409616470337, -0.7042360305786133, 0.5009320378303528, -1.0802503824234009, 0.5784285068511963, -0.4671440124511719, 0.9002493619918823, -0.7876971364021301, 0.6658058166503906, 0.2604207694530487, -0.6095471382141113, -0.23867793381214142, -1.5515302419662476, 1.7713489532470703, 0.17238906025886536, -1.9036365747451782, 0.6279038786888123, -2.3993124961853027, -1.3167394399642944, 0.5757833123207092, 1.0836818218231201, 0.21319229900836945, -0.7733177542686462, 0.4181009531021118, 0.9144922494888306, -0.34190624952316284, -0.9058122038841248, 0.5234060883522034, -0.26726993918418884, -0.65550297498703, -0.43582332134246826, -1.8038170337677002, -0.050222184509038925, -1.9281678199768066, 0.5071417689323425, -1.0329620838165283, -0.27290886640548706, -0.7496196031570435, -0.948323130607605, 0.351288765668869, 0.39388781785964966, 0.6577332019805908, -0.6622164249420166, 1.5166791677474976, -0.04793669283390045, -0.027072688564658165, 0.4490283131599426, -0.11833763122558594, -0.6511051654815674, 1.5550588369369507, 0.9673696160316467, 0.745314359664917, -2.033691644668579, 0.25884321331977844, 0.71192467212677, -0.22104892134666443, -2.0914623737335205, -0.6676270365715027, 0.14032600820064545, -1.2978808879852295, -2.8957529067993164, -0.26426345109939575, -0.18819892406463623, 1.1758854389190674, -0.6133952736854553, -1.3403767347335815, -0.18888433277606964, 1.1060677766799927, 0.002659468911588192, 1.0760074853897095, -0.2461884766817093, 1.0484367609024048, -0.05656522884964943, 0.5097987651824951, -0.5714720487594604, -1.9403458833694458, -1.6080490350723267, -1.1172513961791992, 0.14509496092796326, 0.196913480758667, -1.6546357870101929, 0.3180813789367676, -0.49701398611068726, 0.016535846516489983, 1.5963387489318848, 0.6909794807434082, 0.48499003052711487, -0.5911821126937866], [-1.2864302396774292, -0.9406098127365112, -1.4081937074661255, -0.4225725829601288, -0.16037523746490479, -0.7642636895179749, -0.6086977124214172, 0.037056922912597656, -0.5394694805145264, 0.5111126899719238, -0.6071351170539856, 0.5707187652587891, 0.02482716366648674, -1.4478923082351685, 0.9467079043388367, -0.48045361042022705, -0.8666024804115295, -1.4573209285736084, -0.31104227900505066, 1.0270287990570068, 1.311374545097351, 1.4846956729888916, -0.5016567707061768, -0.04131642356514931, -0.05159076675772667, -2.049779176712036, 0.4856836497783661, 0.2698514461517334, 0.3679162561893463, 0.0959504172205925, 0.7118039727210999, 0.045020926743745804, 0.34949588775634766, 0.8595196604728699, 0.49537548422813416, -0.340729683637619, 1.2497680187225342, 0.5740764141082764, 0.7488643527030945, -0.02777576632797718, 0.2959607243537903, -2.903385639190674, 0.44045716524124146, 1.4665559530258179, 0.16026023030281067, 0.8386685848236084, 0.7827854752540588, -1.358036994934082, -0.49160680174827576, 0.05529213324189186, 0.17847943305969238, 0.19826090335845947, 0.6071869134902954, 0.5410505533218384, -0.5733139514923096, -1.4977461099624634, 0.18526414036750793, -1.0083014965057373, 1.0161415338516235, 0.7687555551528931, 0.8737677931785583, 1.598211646080017, -0.11598518490791321, 0.6691564321517944, -0.8937496542930603, -0.31076228618621826, -0.4050871431827545, 0.19113120436668396, 1.1056156158447266, 0.04369061440229416, 1.5515583753585815, 1.0592817068099976, -0.4209462106227875, 0.28346461057662964, -0.6227483153343201, -1.311898112297058, 1.7294862270355225, 1.3130630254745483, -0.2389722764492035, 0.8901301026344299, 0.1724328100681305, 0.5946865081787109, -0.4193553924560547, -0.14908570051193237, -1.2673214673995972, 0.17460300028324127, 1.3396002054214478, 0.3364273011684418, 0.5155758261680603, 2.190211534500122, -0.4339239299297333, 0.15649394690990448, 0.08952793478965759, -1.054916501045227, -0.3817404508590698, -0.6608672738075256, -0.5820518732070923, -2.7778918743133545, 0.9372061491012573, -0.6124703288078308, 0.7488695383071899, 2.1304523944854736, -0.37112951278686523, 0.730506420135498, 0.8146321773529053, 0.9533789753913879, -1.1133878231048584, 1.4493108987808228, -0.013292200863361359, -0.5967431664466858, -0.4696943759918213, 0.12659424543380737, 1.0240685939788818, 0.7103468179702759, -1.2803921699523926, 0.583183228969574, -0.3200801610946655, 0.9025595784187317, 0.9498392939567566, 1.4992938041687012, -0.783802330493927, -0.4266643822193146, -0.2961026430130005, -0.28030166029930115, 0.8545295596122742, -0.13517065346240997, 1.036967396736145, -0.4602167308330536], [0.8000430464744568, 0.27760905027389526, 1.508948564529419, -0.4119258224964142, 0.7037650346755981, 0.09798896312713623, 1.2264440059661865, 0.15942437946796417, 0.6580092906951904, 0.7708291411399841, 2.061475992202759, -0.31184518337249756, 0.4572663903236389, -0.18997624516487122, 0.5182398557662964, 0.29010888934135437, 1.0494208335876465, -0.3734950125217438, -0.13723042607307434, -2.380080223083496, 0.4618620276451111, 0.1940508633852005, 1.8496142625808716, 0.16696682572364807, 0.648853600025177, -0.21553993225097656, -1.8371264934539795, -0.6659364700317383, 0.8978442549705505, 0.1576402485370636, -1.0758531093597412, -2.096011161804199, -0.017565740272402763, -0.8067179322242737, 1.043199062347412, 0.053477492183446884, -0.5813193917274475, 0.5092154741287231, -0.8421168327331543, 1.1424368619918823, -0.3395845592021942, 0.18121880292892456, -1.3771703243255615, 1.2059733867645264, 3.4698381423950195, 0.8579230308532715, -0.7624855041503906, 0.5087942481040955, 0.4918537735939026, 1.3123326301574707, 1.111075520515442, -0.9948127269744873, 1.2096792459487915, -1.2651475667953491, 0.1160687506198883, 1.2779031991958618, -1.0948984622955322, 0.2652271091938019, -1.6708918809890747, -0.7454444169998169, 1.1384408473968506, 0.8083745241165161, -0.6399542093276978, 0.6185500025749207, -0.41176852583885193, 0.23130109906196594, -0.4069705903530121, 0.6400631070137024, -1.917864441871643, -1.4939852952957153, -0.6931261420249939, 0.4643881618976593, -1.5124554634094238, 0.7462847232818604, 1.8415008783340454, 0.32364794611930847, 0.07564910501241684, -0.7643004655838013, 0.33709052205085754, -0.12622854113578796, 1.8746308088302612, 2.454277276992798, 0.41219794750213623, 0.4580122232437134, 0.6732673645019531, -0.5325409173965454, -1.5613322257995605, 1.0423978567123413, -0.8970410227775574, 0.06962364912033081, 0.3006797134876251, 0.4430079162120819, 0.2709527909755707, -0.0138546172529459, 0.4316141903400421, -0.40161287784576416, 0.2370879352092743, -0.3876463770866394, 1.9110138416290283, 1.7360270023345947, -0.21585038304328918, 0.14967699348926544, 0.021703410893678665, -0.8693311810493469, -0.3485683500766754, -0.319022536277771, -0.5443957448005676, 1.5563184022903442, -0.0700889378786087, 0.37668803334236145, 2.132654905319214, 0.3249224126338959, 1.7948237657546997, -0.6841837763786316, -0.7516117095947266, 0.2384064793586731, -0.8069108724594116, 0.3553768992424011, -0.69162917137146, -0.6341233253479004, 0.3641471862792969, -0.29559528827667236, -0.7998982071876526, -1.3764009475708008, -1.2838844060897827, -0.6596519351005554, 1.9768046140670776, 0.1640027016401291], [-0.03680144622921944, 1.0127612352371216, 1.826758861541748, 0.20749643445014954, 1.4535082578659058, -0.28950396180152893, 1.1633051633834839, -0.08226149529218674, 1.4521468877792358, 0.2616744041442871, -0.1520678699016571, 1.4680125713348389, -0.4903222322463989, -0.05612756311893463, 0.6609936952590942, 1.4089996814727783, -0.3087816834449768, 0.891100287437439, 0.049567848443984985, -1.0178263187408447, -0.6602705717086792, 0.08753082901239395, 1.4768824577331543, 1.1653919219970703, -0.3293251097202301, -0.72516930103302, 2.2402822971343994, 1.170725703239441, -1.3065012693405151, -0.011027291417121887, 1.9080989360809326, 1.5802031755447388, -1.1785614490509033, -2.5467238426208496, 0.9235764145851135, -1.294083833694458, -0.8318848013877869, 0.35246509313583374, 1.759677529335022, 1.0793488025665283, -0.11365730315446854, 0.5296629667282104, -1.0109020471572876, -0.4591560959815979, 0.5893855690956116, 0.03294515982270241, 0.5216041803359985, -0.594308078289032, 1.4723231792449951, 0.2171829640865326, 1.5039443969726562, 2.7655398845672607, -0.8244000673294067, 0.28543350100517273, -0.3678343892097473, -0.3366933763027191, -0.10773767530918121, 0.5841479301452637, 0.04500773921608925, 0.07286377996206284, 0.5927274823188782, -1.3265494108200073, 0.04044903442263603, -0.22041063010692596, -1.8741153478622437, 0.6116737723350525, 0.47209444642066956, 1.180503487586975, -0.20428383350372314, -1.4836490154266357, -0.4796389639377594, 0.8638951778411865, 0.45855921506881714, -0.2712059020996094, 0.961409866809845, 0.14289548993110657, -1.541872262954712, -2.0131444931030273, 0.28858762979507446, -0.44901418685913086, -0.1740923523902893, 1.085089087486267, 1.431462287902832, 1.8570940494537354, -0.5490501523017883, -1.4896866083145142, -1.8135955333709717, 0.12615616619586945, -2.084981918334961, 0.2274060696363449, -1.048155665397644, -0.34074679017066956, 0.4626527428627014, -1.4233975410461426, -0.2467958778142929, -0.7340630888938904, -0.9955815672874451, 0.7685922384262085, -0.5253556966781616, -0.230552539229393, 0.1745767742395401, 1.7160793542861938, 0.7581731677055359, 1.092392921447754, 0.39744794368743896, 0.3446493148803711, 0.8909921646118164, -1.7649637460708618, 0.4972265064716339, 0.5704648494720459, -0.14481571316719055, -0.8576288819313049, 0.4675464332103729, -1.7551194429397583, 1.6096606254577637, 0.9712414741516113, -0.5374245643615723, -0.16471995413303375, -0.4631302058696747, 1.3128551244735718, 1.1917487382888794, -0.33345481753349304, 0.24109824001789093, -0.2972831428050995, 0.41933175921440125, -0.08420398831367493, -0.4432164132595062, 0.9568894505500793], [0.1044735461473465, -1.1887294054031372, -0.29320842027664185, -0.9873883128166199, -1.3531900644302368, 0.16353081166744232, 0.4551786482334137, 0.16102391481399536, -1.2501219511032104, 1.1046169996261597, 0.9316409230232239, -1.760475754737854, 0.23921947181224823, 0.7302502393722534, 0.5314520001411438, 1.9371851682662964, 1.2952324151992798, -1.0635885000228882, 0.6193556785583496, -2.35015869140625, -0.2919919788837433, 0.6921700835227966, -1.1822340488433838, 0.19340723752975464, 0.1298888921737671, -2.208752393722534, -0.10845772176980972, -0.5176720023155212, 0.6628892421722412, 1.009392499923706, -0.7938235402107239, 1.0617929697036743, -0.31182152032852173, -1.4618020057678223, -0.2517050802707672, 0.1976802498102188, 1.280371904373169, 0.3465060889720917, 1.2619378566741943, -0.9957333207130432, 0.9208282828330994, -0.6428104043006897, 1.3372571468353271, -1.4581528902053833, 0.45714718103408813, 0.24817989766597748, -0.4555528163909912, 1.3396669626235962, 1.041678786277771, 0.2599967420101166, -0.10593234747648239, 1.819183111190796, -0.5428086519241333, 0.9346213936805725, 0.11561909317970276, -0.06490075588226318, 0.21812336146831512, -0.5983142852783203, -1.2687698602676392, 0.46883612871170044, 1.057028889656067, 0.43933799862861633, 1.0390357971191406, 1.5962361097335815, 1.3887277841567993, -0.037384845316410065, 0.4432354271411896, 0.22928182780742645, -1.3626213073730469, 0.9377244710922241, 1.311203122138977, 2.5455117225646973, 0.1919122189283371, 0.31976839900016785, 0.21122565865516663, 1.871048092842102, 0.046977538615465164, -1.505743145942688, 0.09408116340637207, 0.3699873387813568, 0.36842504143714905, -0.6800699830055237, 1.3835086822509766, -1.9397902488708496, 0.0386895090341568, -1.982666254043579, 0.7983017563819885, 0.08301515132188797, 1.8298112154006958, -0.22654391825199127, -0.5705928802490234, 0.20068003237247467, 0.5407873392105103, 1.4317216873168945, 2.240668773651123, 0.4356084167957306, -1.0807135105133057, 0.006371916737407446, 1.1122974157333374, 0.943434476852417, 1.4361451864242554, 1.0145899057388306, 0.560133695602417, 0.5215003490447998, 0.6633070111274719, -0.8972636461257935, 1.6029764413833618, -0.5453208684921265, -0.5030078291893005, -0.04998871684074402, -0.7124826908111572, -0.11002087593078613, -1.0250765085220337, 0.1859419345855713, 0.208116814494133, -0.4527830481529236, 0.7626046538352966, 0.42417770624160767, -0.5916182994842529, -1.221064567565918, 0.9166616797447205, -0.05242885276675224, -0.7294141054153442, -0.4672509431838989, 0.6925830245018005, 1.1179540157318115, -0.08518385142087936, -1.0348588228225708], [-0.2686121165752411, -0.7795184254646301, 1.366607666015625, 0.735192060470581, -2.0275652408599854, -0.3016469180583954, 0.77790766954422, 0.16985034942626953, -0.15934157371520996, 1.029398798942566, 0.6653238534927368, -0.16824547946453094, -0.2464239001274109, -0.5407261252403259, -0.19145239889621735, -0.003732770448550582, -0.1263531893491745, -1.2026792764663696, 0.4514175057411194, 0.8203341364860535, -0.10054576396942139, -0.339758038520813, -1.417688012123108, -1.0492233037948608, -0.8674338459968567, -0.7352785468101501, -0.8645598888397217, -0.6553235650062561, -0.13031332194805145, -2.230103015899658, -0.02294595167040825, -1.2632603645324707, -0.1566299945116043, -1.5771453380584717, -1.2240960597991943, 1.4409959316253662, 0.44941484928131104, -0.9354802966117859, 0.020193321630358696, 0.4346346855163574, 1.0501127243041992, -0.07445865124464035, 1.0081154108047485, -0.2716182768344879, -0.5034066438674927, -0.7941250801086426, 0.9245307445526123, 0.15846110880374908, -1.647341251373291, 1.4726066589355469, 0.054237473756074905, 1.6938472986221313, -0.24230380356311798, 0.705544650554657, -0.7971477508544922, -0.5928152799606323, -0.2944321036338806, -0.07767564803361893, 1.392551064491272, 0.8426432013511658, -1.08740234375, 0.4710662364959717, 0.27378755807876587, 0.47632357478141785, -1.591805100440979, -0.4525567889213562, -0.1909802109003067, -0.3094368278980255, 0.6815810799598694, 0.8255165815353394, -0.14425227046012878, 0.4701383709907532, 1.3360586166381836, -0.3847719430923462, -0.39122194051742554, -0.12580150365829468, -0.46242743730545044, 1.795100450515747, -0.9228523373603821, -0.757012665271759, 0.9441424608230591, 1.0982805490493774, 1.3473237752914429, -0.942895770072937, -0.8716007471084595, -0.16464826464653015, -0.9507518410682678, -0.4664331376552582, 0.47486093640327454, -1.617594599723816, 1.4233372211456299, 0.015405837446451187, -1.7821109294891357, 1.477927803993225, -1.2034039497375488, -0.510844886302948, -0.5007436275482178, 0.045571424067020416, -0.32280582189559937, 1.2640399932861328, -1.007485270500183, 0.5183030366897583, 0.3972185254096985, -0.009841457940638065, 1.1390104293823242, 0.36660680174827576, -0.22492846846580505, -0.11278428137302399, 0.9900575280189514, 2.85477352142334, -1.045589566230774, -0.4327608048915863, 0.33710378408432007, -0.6072806715965271, -1.6655645370483398, 0.9432735443115234, 1.087454080581665, 0.8800973296165466, 0.37921416759490967, -0.9439065456390381, -0.4090310335159302, -1.306974172592163, -1.4607810974121094, 1.969543218612671, 1.7173291444778442, -0.5242658853530884, -1.397063136100769, 0.10810552537441254], [0.2166830450296402, 0.7852283716201782, -0.028223440051078796, 0.2312713861465454, 0.21816352009773254, 2.8058056831359863, 0.6657280921936035, 0.4672398865222931, 0.15358039736747742, 1.2671263217926025, 0.2083740532398224, -1.1231400966644287, -1.653609275817871, -0.9142298102378845, 0.6334742307662964, -0.7195965647697449, -0.46102970838546753, -0.6589841246604919, -0.8927674889564514, 0.35748037695884705, -0.4490991234779358, 1.5541324615478516, -1.711318850517273, -0.04357537627220154, 0.8636502623558044, -1.0068353414535522, -1.5978186130523682, 1.7446905374526978, 0.37426263093948364, 0.7453216910362244, 0.017948660999536514, 1.0572699308395386, -0.25745806097984314, -1.5745564699172974, 1.7805346250534058, 1.0047811269760132, -0.06884465366601944, 0.9985597133636475, 1.1611826419830322, -0.6509393453598022, -0.8714408278465271, -0.4801764190196991, -1.775210976600647, -0.48133495450019836, -1.4429727792739868, -0.49759137630462646, 0.8325718641281128, 0.8260226249694824, -1.010166883468628, -1.464241862297058, -0.37101849913597107, -0.3581041395664215, 0.13067412376403809, -2.148422956466675, -0.5053144097328186, 2.3465535640716553, 1.6601086854934692, 1.0358082056045532, 0.15019434690475464, -0.9120731949806213, -0.030288297683000565, 0.9536609053611755, -1.370017409324646, 0.5486972332000732, 0.34368789196014404, -1.1450507640838623, 0.11422330141067505, 0.09399097412824631, -0.09323009848594666, -1.4439632892608643, -0.36108195781707764, -0.2100965827703476, -0.4469265639781952, -0.7980807423591614, 0.5338172316551208, -2.217121124267578, 0.48634886741638184, -0.47007474303245544, 1.362432837486267, 1.6873602867126465, 0.19423413276672363, -0.18987305462360382, 0.6279640793800354, -0.6621710062026978, -0.6411495804786682, 0.5911665558815002, 0.21644842624664307, 1.04438316822052, 0.5590285062789917, 0.3952482342720032, -1.6319024562835693, 1.0738179683685303, 0.7877583503723145, 1.7400683164596558, -0.5446556210517883, 0.361853688955307, 0.9014566540718079, -0.7308380603790283, 1.871389389038086, -1.3961693048477173, 0.7260664701461792, 0.8357518911361694, -0.8575359582901001, -0.25747841596603394, -0.40005314350128174, 0.6964988708496094, 1.4556437730789185, -1.554720163345337, 1.1948168277740479, 1.4597631692886353, -1.32685124874115, 0.6211540102958679, -1.4868375062942505, 0.9727198481559753, 1.979323148727417, 0.9039032459259033, -1.1162716150283813, -1.6475030183792114, -0.5008391737937927, 0.12761688232421875, -1.060876488685608, -0.40129464864730835, 0.7690796852111816, -0.4720304310321808, -2.8624167442321777, 0.5423724055290222, -1.098728895187378, -0.13106995820999146], [-1.7919062376022339, -0.5469270348548889, -1.4449396133422852, 0.015298665501177311, 1.0025906562805176, 0.5503202676773071, 0.5328980088233948, 1.0890792608261108, 0.17375294864177704, 0.9968735575675964, 1.3992455005645752, -1.0323798656463623, -0.09257661551237106, 0.38347673416137695, -1.5154354572296143, 0.5136014223098755, 0.2282601147890091, 0.574799656867981, -2.375437021255493, 0.4488210678100586, 0.14259524643421173, -0.9807693362236023, 0.17440113425254822, 1.970819354057312, 0.15677912533283234, -0.329256147146225, 0.9517865180969238, -0.30666297674179077, -3.1389963626861572, -0.08712553977966309, -0.21820484101772308, 1.315581202507019, 1.1650251150131226, 0.24153392016887665, 0.6998282074928284, 0.04242662712931633, -0.125960573554039, -2.379692792892456, 0.02777414582669735, -0.29734504222869873, -0.3614511787891388, -0.20444844663143158, -0.42998620867729187, 1.4472728967666626, 0.3039562404155731, -2.271684169769287, -1.3051273822784424, -1.3373970985412598, -0.325954407453537, 0.06604789942502975, -0.41696009039878845, -0.6183850765228271, -0.21159963309764862, 0.6080316305160522, -0.808502733707428, 1.3286620378494263, -1.3589104413986206, 0.9918205738067627, -0.9798195958137512, 0.012839125469326973, -1.1938401460647583, -1.4946775436401367, -0.9309188723564148, 0.16570572555065155, -1.4601707458496094, -0.9709855318069458, -0.8101339936256409, -0.139214426279068, -0.8380381464958191, 0.42141610383987427, 0.8922685980796814, -0.22539974749088287, 0.7815411686897278, 0.34229499101638794, 1.1109108924865723, 0.3662642538547516, 0.7147271037101746, 0.4402858018875122, 0.9637870192527771, -0.5057390332221985, -0.20247980952262878, -1.033969759941101, -0.21989397704601288, 0.4507353901863098, -0.254706472158432, 0.169633150100708, -1.2044627666473389, -0.47166723012924194, -0.08874329924583435, 0.19864152371883392, -1.039511799812317, -0.10299371182918549, -1.018547773361206, 0.7192819118499756, -0.4369468688964844, -1.3792635202407837, -1.5569406747817993, -0.2695971131324768, 0.23428666591644287, -0.15985332429409027, -1.3738458156585693, -1.7443283796310425, -0.7403014302253723, -0.7409083843231201, -0.24065586924552917, 0.6912752985954285, -2.5626862049102783, -0.4697657823562622, -0.8200021386146545, 0.48987266421318054, 0.37036848068237305, -0.7122719287872314, -0.30992019176483154, 1.1256321668624878, 1.529478669166565, 0.1870279163122177, 0.13414455950260162, 0.19706055521965027, 1.2562767267227173, -0.37913838028907776, 4.358133792877197, 0.14161324501037598, -0.6868207454681396, 2.2908153533935547, -0.26267027854919434, 0.5162164568901062, -0.40625452995300293, -1.913337230682373], [-0.6442956924438477, -0.21780551970005035, -1.8166989088058472, -0.7839356660842896, -1.3607409000396729, -0.01519079227000475, 0.03238876909017563, -0.06875608116388321, -1.6069945096969604, -0.2456761747598648, -1.2444325685501099, -1.010869026184082, -1.3981610536575317, 0.18366526067256927, 0.7882739305496216, -1.3194029331207275, -0.41309455037117004, -0.12827780842781067, 0.5876232981681824, -1.1422858238220215, -0.4294085204601288, 1.3193886280059814, -0.3207540512084961, 0.2267860770225525, 1.1086817979812622, 0.4788864254951477, 0.43403586745262146, 0.2166789025068283, -0.7275040745735168, 1.483012318611145, -0.9192080497741699, -0.22000351548194885, 1.051640510559082, -2.610808849334717, 0.48905089497566223, 0.46124401688575745, -1.0008968114852905, -0.7571125626564026, 0.28867417573928833, 0.0936337262392044, -0.2796977758407593, -0.7399749159812927, -0.4791162610054016, -0.11240930110216141, -0.4237707555294037, 0.06341809034347534, 0.31183913350105286, -2.3125429153442383, 0.6770647168159485, 2.877601385116577, -0.3529120981693268, 1.0502134561538696, -1.0186368227005005, -0.6254823803901672, 0.3869604170322418, -1.0875502824783325, -0.10080286115407944, -1.1687582731246948, 0.5361711382865906, 0.7621702551841736, 0.42221933603286743, -0.4338396191596985, -1.2849870920181274, -0.2196294665336609, 1.3411855697631836, -1.6315070390701294, 0.45416945219039917, -2.4147861003875732, -0.9286304116249084, -0.7865217328071594, 0.5019314885139465, 0.3003973066806793, -1.1521618366241455, 0.07016649842262268, 0.055032238364219666, 0.25798285007476807, 0.9226936101913452, -0.5848655104637146, 0.40382009744644165, 0.38695451617240906, 2.2159993648529053, 0.9493293762207031, 1.0321545600891113, -0.9602651596069336, 0.7933881878852844, -1.3416591882705688, 1.6147048473358154, 1.445939064025879, -1.6706304550170898, -1.8234022855758667, 2.096172571182251, -0.959614098072052, -1.0751779079437256, -0.3973865807056427, -1.1554282903671265, 0.283424437046051, -0.13702338933944702, 0.8748270273208618, -1.7686715126037598, -1.9446600675582886, -0.27626386284828186, -0.6997452974319458, -0.20023402571678162, -2.0537497997283936, 1.7321579456329346, 1.4275318384170532, -0.15766789019107819, 1.5272222757339478, 0.12494351714849472, 1.3317856788635254, 0.5147797465324402, 2.7960429191589355, -1.0573118925094604, -1.0903133153915405, -0.6471196413040161, -1.0352131128311157, 2.7064950466156006, 0.1111505925655365, 0.6780814528465271, 0.9539521336555481, 0.866995632648468, -2.0597591400146484, 1.1197282075881958, 0.13912194967269897, -1.038486361503601, -0.7125319242477417, -0.10056470334529877, 1.3746697902679443], [-0.5123711228370667, 0.377164363861084, -1.05585515499115, -0.8448875546455383, -0.4349973201751709, 0.8395900130271912, -0.4618307054042816, 0.2603023648262024, 1.0098106861114502, -1.4819995164871216, -0.30681338906288147, -1.215267300605774, -1.0276691913604736, 1.0081645250320435, 0.3367028832435608, 0.19741986691951752, -0.955866813659668, 0.1511666476726532, 1.3646217584609985, -0.68101966381073, 1.3347960710525513, -0.19805549085140228, -1.3768880367279053, 0.7361053228378296, -0.11955147981643677, 0.783743679523468, -1.4822416305541992, -0.4483334720134735, -0.18182332813739777, 1.1555626392364502, -1.955976963043213, 1.4130029678344727, 0.3337439298629761, -1.6328697204589844, 0.28083541989326477, -0.05860557779669762, 0.3561864197254181, 0.7643469572067261, 0.5774955153465271, 0.36410585045814514, 0.6573681235313416, -1.2302440404891968, -0.37885692715644836, -0.5117335915565491, 0.10903949290513992, 2.115699291229248, -0.6071828007698059, -0.7295584678649902, -1.411082148551941, -1.4770827293395996, -0.0835454985499382, -0.33179688453674316, 0.591291069984436, -0.6756489872932434, 0.5614211559295654, 0.10351287573575974, 0.45852720737457275, -1.4763352870941162, -1.8429380655288696, 0.12189700454473495, -0.587013840675354, -0.7967780828475952, 0.7259436845779419, 0.8412687182426453, -0.6333652138710022, 0.3521486818790436, -1.342392086982727, -1.0256863832473755, 0.5040200352668762, 0.7941274642944336, 1.0975617170333862, 0.8818901181221008, 0.622410237789154, 0.28080886602401733, 0.3622702360153198, -1.389165997505188, -0.6007702350616455, 0.18799223005771637, -0.056085046380758286, -1.1021960973739624, 0.6567197442054749, -2.1239964962005615, -0.8670592904090881, -0.04373683035373688, 1.3714596033096313, 0.6781570315361023, 0.13625529408454895, 1.5498034954071045, -1.3569954633712769, -0.16050752997398376, 0.67720627784729, -0.5919163227081299, -0.38170763850212097, 0.843325674533844, 0.5832788348197937, -2.7347755432128906, 0.23293690383434296, 1.0024610757827759, 0.19949814677238464, -0.37068048119544983, 1.1152430772781372, -0.27580544352531433, 0.18406066298484802, -0.8042879700660706, 0.08291325718164444, -0.5249057412147522, -0.35191553831100464, 0.017810596153140068, -0.19588002562522888, -1.2087254524230957, 0.8301162719726562, -0.6224959492683411, 1.5300318002700806, 0.47129830718040466, 0.6923683881759644, -0.441962331533432, -1.173007845878601, 2.4263083934783936, 0.5301488637924194, -0.10874083638191223, -0.3848208487033844, 1.7849912643432617, 0.4580475389957428, -1.896314024925232, 0.5701732039451599, -0.5553039908409119, 0.8187189102172852, -2.031209707260132], [0.33052894473075867, -0.15432661771774292, -0.24266153573989868, 0.4499741196632385, -0.22977110743522644, 0.7287571430206299, -0.46770742535591125, 0.39100906252861023, 0.3095221519470215, -0.5219412446022034, 1.4171279668807983, 0.5669938921928406, -0.6583457589149475, -0.1035190224647522, 0.8391349911689758, -0.4020247459411621, -0.14360244572162628, 0.35508233308792114, 0.7622880935668945, 0.11596830934286118, 0.15539366006851196, -0.9514496922492981, 0.06284810602664948, -1.268528699874878, 0.140313982963562, 0.3002067506313324, -0.07708968222141266, 0.8627916574478149, 0.4120013117790222, -0.15354984998703003, 0.13321994245052338, -1.3691874742507935, -1.3696242570877075, 0.6079028844833374, 1.1666139364242554, -0.6463244557380676, -0.6681636571884155, -0.5195295810699463, -0.9919540882110596, -1.4520275592803955, -0.2023569643497467, 0.38493797183036804, -0.8264586329460144, 0.9202795028686523, 0.19239921867847443, 0.17621760070323944, 0.4309242069721222, -0.37299346923828125, -0.16764676570892334, 0.1977594643831253, 0.12444322556257248, 1.7989178895950317, -0.1953561156988144, 0.5920642614364624, -1.1094844341278076, -1.5919008255004883, 1.0816549062728882, -0.2229810655117035, 0.31113600730895996, 1.3165684938430786, 0.1428278088569641, -0.4758191704750061, 0.7228778004646301, -1.1418828964233398, 0.42039579153060913, -0.7936484813690186, -0.06268736720085144, -1.0206037759780884, -0.11483107507228851, -0.5597440600395203, -1.2780567407608032, 0.2058582305908203, -0.7836376428604126, -0.4974232614040375, -0.6355326771736145, 0.781655490398407, -0.27427220344543457, -0.5256569981575012, 1.6782835721969604, -0.020349331200122833, -0.4701111614704132, -1.2183129787445068, -1.1194311380386353, 1.2687033414840698, 2.273837089538574, 1.64288330078125, -1.0150560140609741, 0.8453081250190735, -0.5814616680145264, -1.1262497901916504, 0.23205874860286713, 0.028834104537963867, -0.013582469895482063, -0.3184361755847931, 0.72402024269104, -0.7335285544395447, 1.9702718257904053, -0.4004638195037842, -0.7077096700668335, -0.0449974462389946, 0.8376877903938293, -1.0170481204986572, 1.5491971969604492, -1.0343092679977417, -0.22040142118930817, 0.8929651379585266, -0.3948797583580017, -0.4295700788497925, 0.7081305980682373, 0.5386171936988831, -0.8158568143844604, -0.3628467321395874, 0.12685935199260712, 1.607357382774353, 0.09417083859443665, 1.9615181684494019, -2.1867215633392334, -0.23632435500621796, -1.2618387937545776, 0.22518888115882874, -0.6157538890838623, -1.7526273727416992, -2.1253726482391357, -1.3497124910354614, 0.49156662821769714, 1.7537440061569214, -1.4340827465057373, 1.2482268810272217], [-0.12577062845230103, 1.577799916267395, 0.8423697352409363, 0.629983127117157, -1.8503236770629883, 1.5299180746078491, -1.611750841140747, 0.723257839679718, -0.6251975297927856, 0.4769235849380493, 1.3992953300476074, 0.6810898780822754, 0.44481876492500305, -0.23024457693099976, -0.9428313970565796, -0.3297199308872223, 1.9606307744979858, -0.5955936908721924, -0.6379733681678772, 0.44709211587905884, 1.074293851852417, 0.08450436592102051, -2.996474504470825, 1.9006047248840332, -0.6226083636283875, -1.0296448469161987, -0.6470330357551575, -0.9241832494735718, 0.31338831782341003, -0.31258276104927063, 2.1432507038116455, -1.346193790435791, -0.2872285842895508, -0.20773640275001526, 1.0136158466339111, 0.9946075081825256, 2.008525848388672, -0.07368157058954239, -0.25179314613342285, -0.7120784521102905, -0.07838686555624008, 0.03158712759613991, 1.1821848154067993, -1.0739920139312744, 0.7763966917991638, -2.798308849334717, -1.015495777130127, -1.7113287448883057, -0.9410465359687805, 0.29851871728897095, -1.485756278038025, -2.6138715744018555, 0.8697222471237183, -0.8542872667312622, 3.2167160511016846, -0.5789056420326233, -2.1478867530822754, 0.8378142714500427, 0.3036109209060669, 0.927475094795227, -0.10890071839094162, 0.057704050093889236, -1.3878110647201538, -0.5784735083580017, 0.7844312787055969, -0.5832169055938721, -2.131136417388916, -0.3890073597431183, -0.09466011822223663, 0.5531739592552185, 1.2076611518859863, -0.12431258708238602, -0.15200190246105194, 1.0561296939849854, 0.4265725016593933, -0.3292691707611084, -0.5850570797920227, 0.51677405834198, -0.8089162707328796, 0.044228654354810715, 0.9063275456428528, -1.4918241500854492, 0.3315511643886566, 0.6216435432434082, -0.9808988571166992, -0.13539093732833862, -0.08806106448173523, -0.47608116269111633, -1.085270881652832, 0.5959898233413696, 1.8392462730407715, -1.9563649892807007, 0.057748015969991684, -0.4410327970981598, 0.36342546343803406, 0.38625818490982056, 0.15089058876037598, -0.9426610469818115, -0.7157153487205505, -0.3037523329257965, -2.7017247676849365, -1.2678302526474, 1.6986987590789795, 0.4631045162677765, 0.6762595772743225, 1.6225547790527344, 0.1974411904811859, 0.9703807830810547, 0.48987308144569397, -0.9110308289527893, -0.7835853099822998, -0.4844730496406555, -0.9891896843910217, -1.195419430732727, -1.810361623764038, 0.9345821142196655, 0.26511165499687195, -0.5153065919876099, -0.12090184539556503, 2.969571352005005, 0.43406882882118225, 0.06815164536237717, -0.42782849073410034, 0.3132409453392029, 1.2681491374969482, -0.43678557872772217, 0.9384967088699341, -0.45496997237205505], [1.3803631067276, -0.4516138732433319, -0.2302066534757614, -1.094677448272705, -0.022660991176962852, -0.526973307132721, 1.1535379886627197, 0.9860732555389404, 0.06098487600684166, -0.38924551010131836, -0.645829975605011, 1.300688624382019, 0.4298871159553528, -0.3650353252887726, -1.3552520275115967, 0.11794818192720413, 0.1669822484254837, 0.3406546115875244, 1.3136054277420044, -0.4942464828491211, -0.522464394569397, -0.34548935294151306, -0.2608213424682617, -1.275165319442749, -0.4260387420654297, 0.37981116771698, -1.3832197189331055, 0.49885278940200806, 1.6186363697052002, -0.31211647391319275, 0.4650548994541168, 0.7027717232704163, -0.859296977519989, -0.7505664825439453, 0.795342206954956, 0.7488452792167664, -0.1669432371854782, 0.25031259655952454, 0.7649753093719482, 0.40321657061576843, 0.12569406628608704, 0.055658210068941116, -1.421391487121582, -1.1377657651901245, -0.181367889046669, 1.7814418077468872, 0.6141119003295898, -0.7809141278266907, 0.014259307645261288, -0.6874573230743408, -0.7541237473487854, 1.5824205875396729, 0.8208954930305481, -1.4118835926055908, 0.7899361848831177, -0.5345852971076965, 0.29148969054222107, 0.040717776864767075, 0.15194913744926453, -1.2894989252090454, 0.9632729291915894, 1.5364536046981812, -1.1764488220214844, 1.4991179704666138, -0.7108094096183777, -0.5184676051139832, -0.21720659732818604, 0.6987720727920532, 1.137762427330017, 0.6943453550338745, 0.2733825743198395, 1.2374681234359741, -1.653228998184204, -0.6021995544433594, 1.6412959098815918, 0.13519813120365143, 0.31875836849212646, 0.21446514129638672, -0.11261824518442154, 0.6481477618217468, -1.1357837915420532, -0.4493468701839447, 1.2485052347183228, 0.7447147965431213, -1.1877800226211548, -1.6814175844192505, -0.4603725075721741, 0.11734580248594284, -0.1584014743566513, -1.218934416770935, -0.42483705282211304, -0.36378392577171326, -0.6933138966560364, -0.08901365846395493, -0.9684531688690186, 0.7352501153945923, -0.4305914342403412, 1.3748711347579956, 0.5294771194458008, 1.7501108646392822, 0.315012127161026, -0.2652207016944885, -1.0394046306610107, -0.43227386474609375, 1.7126550674438477, 1.6850743293762207, -0.47191014885902405, 0.7882295250892639, 0.11513347923755646, 0.014608683995902538, -0.5145408511161804, 1.0942796468734741, -0.17248211801052094, -1.549330234527588, 0.827050507068634, 0.5401696562767029, 1.206424355506897, 0.9669725298881531, 0.7867423295974731, -0.27627134323120117, 0.4845953583717346, 0.19921600818634033, -0.061044495552778244, -0.13457037508487701, -0.1116248071193695, 0.25543802976608276, -0.15277576446533203, 2.0952372550964355], [-0.7749654054641724, 0.35146740078926086, 0.5011723637580872, -1.6853135824203491, 1.1186518669128418, 2.0337510108947754, -0.3894962966442108, 0.406827449798584, -0.6631101369857788, 0.1961287558078766, -1.3093469142913818, -1.7735540866851807, 0.7782742977142334, 0.08945448696613312, -0.6607143878936768, -1.2198303937911987, 1.2262887954711914, -0.8928142786026001, -0.0838143602013588, 0.6254973411560059, 0.7008189558982849, -0.943248450756073, 0.36518770456314087, -0.17978690564632416, -0.9866196513175964, -0.06989417225122452, -0.7221924662590027, -0.2402985394001007, -0.5914836525917053, 0.46510186791419983, -0.4575408399105072, -0.31715744733810425, -0.8314877152442932, -1.3055790662765503, 0.22500838339328766, -0.15258590877056122, -0.0023145766463130713, -0.017410552129149437, -0.92010098695755, 0.3622160851955414, -0.8247863054275513, 0.9624349474906921, 0.652073323726654, -2.0709447860717773, -0.37070411443710327, 2.662895441055298, 0.28382065892219543, 1.272138237953186, -0.06052232161164284, -0.9264238476753235, -0.006626285146921873, -0.2326071858406067, -1.0538246631622314, -0.9458138346672058, 0.8258097767829895, -0.616319477558136, 0.2452450394630432, 0.4721190333366394, -1.7188184261322021, 0.3979859948158264, -1.872463345527649, -1.1561239957809448, 1.3809051513671875, -2.1964170932769775, -0.054859429597854614, -2.5868382453918457, -0.17352959513664246, 0.6270545125007629, 1.185042381286621, -0.12305765599012375, 1.2434372901916504, 2.0693442821502686, 0.027357934042811394, 0.518057107925415, -1.2286564111709595, 0.03863382712006569, -0.8473786115646362, -0.5495759844779968, -0.19043982028961182, 1.6910500526428223, -0.02873031236231327, -0.12342934310436249, 0.7876678109169006, -1.583890676498413, -1.3827320337295532, 1.4092823266983032, 0.7099927067756653, -0.09541887789964676, 0.21714159846305847, 1.7593811750411987, -1.3062337636947632, -0.3037244379520416, -0.00545408995822072, 0.2578364908695221, 0.9073671102523804, 0.46372196078300476, 1.6428899765014648, -0.9184189438819885, 0.35265323519706726, 0.03825832158327103, -0.42941439151763916, -1.5786458253860474, -0.09539609402418137, 0.38499969244003296, -0.4771105647087097, -0.11218473315238953, -0.06335543096065521, 1.4438704252243042, 0.005349131301045418, 0.5511160492897034, -1.70858895778656, -2.610032081604004, -1.5079641342163086, 0.359149694442749, 0.11941425502300262, -0.42034730315208435, -0.9877645373344421, -1.5889827013015747, 0.6932934522628784, -1.9322960376739502, -0.3669322729110718, 1.2622230052947998, 1.7804012298583984, 1.5376051664352417, -0.6099861264228821, 1.8085764646530151, -0.5669037699699402, 0.3402615487575531], [0.059025079011917114, -1.545185923576355, -0.7753914594650269, 1.1166167259216309, -1.291790246963501, 1.0493314266204834, -0.8098201155662537, 1.4849181175231934, 1.566969871520996, -0.7831050753593445, -0.9716253280639648, 0.8098336458206177, 0.6002038717269897, -0.350479394197464, -0.4659936726093292, 1.7904257774353027, -0.02087262086570263, -0.03790861368179321, 1.8302631378173828, 1.0235484838485718, 0.7246776819229126, 0.411985844373703, -1.1374849081039429, -0.8568397760391235, -0.570327639579773, -0.8471822142601013, -0.603232741355896, -0.5280932188034058, -1.130839467048645, -0.6746745705604553, -0.3350105583667755, -0.06838397681713104, -1.6406532526016235, 1.4980350732803345, -1.1582049131393433, 0.9259410500526428, 0.2788398265838623, 1.8865481615066528, 0.3177263140678406, 0.7606648802757263, 0.4846697747707367, 0.2533552348613739, -0.2837820053100586, 0.46364450454711914, 1.7166259288787842, 0.9705795645713806, 0.004535963758826256, -0.6078111529350281, 1.4592559337615967, -1.524067997932434, 0.24130192399024963, -0.42138832807540894, 0.04773362725973129, 0.42175355553627014, -0.8129692673683167, -0.04269026964902878, -1.406259298324585, -0.13595078885555267, -1.3945763111114502, 0.016399694606661797, -1.212312936782837, 1.0199847221374512, -0.37622129917144775, 0.40554144978523254, -1.2675222158432007, -1.3004786968231201, -1.1575504541397095, -0.11192905157804489, -0.42845767736434937, -0.5998931527137756, 0.6592715382575989, -1.1710869073867798, 0.055111464112997055, -1.928680181503296, 0.4654996693134308, 0.6496841907501221, -0.5397921204566956, -0.10435693711042404, -0.5318888425827026, -0.5287817120552063, 0.010555862449109554, 0.376560240983963, -2.011570453643799, 1.8457893133163452, -0.04828004166483879, 0.1976807415485382, 0.3499992787837982, -0.21141430735588074, -0.0434645339846611, 1.4287968873977661, -0.5820686221122742, 1.3271793127059937, 0.7898820638656616, -0.6430345773696899, 0.14570564031600952, -0.8420618772506714, 0.7063876390457153, -0.4342295229434967, 1.3191578388214111, -1.0314332246780396, 2.2929577827453613, 1.9956657886505127, 0.6873042583465576, -1.4574748277664185, 0.03146791830658913, -0.7898858785629272, 0.12914986908435822, -0.44687536358833313, 0.34266701340675354, 1.6658222675323486, -1.0558314323425293, 0.015193913131952286, -1.4369455575942993, -0.8503447771072388, 0.6624135375022888, 0.2447051852941513, -0.3939380943775177, 0.3577035367488861, -0.2772793471813202, -0.2839454710483551, -1.119824767112732, 0.7586902976036072, -1.9939537048339844, -0.3029036521911621, 0.9703022241592407, -0.763725221157074, -0.23079612851142883, 0.01818174123764038], [-0.2545393109321594, 0.462801456451416, -0.475355327129364, 2.6148808002471924, 0.4980788230895996, 1.0520621538162231, 0.3190345764160156, -1.394273042678833, 0.34197816252708435, 0.548385739326477, 0.3681955933570862, 0.5426638722419739, -0.1268349438905716, 0.20786304771900177, 0.023225577548146248, 1.096215009689331, -0.6085333228111267, -0.9331560134887695, -0.4190429449081421, 0.2014576941728592, 0.4396526515483856, 0.8025059700012207, 1.8159801959991455, -0.9021642804145813, -1.095018982887268, -1.2705919742584229, -0.27506858110427856, 0.985052227973938, -0.7445560693740845, -0.547993540763855, 1.0858063697814941, -0.026453226804733276, 1.424223780632019, 2.218508005142212, 0.9442444443702698, 1.5200306177139282, 0.5108605623245239, -0.4925125539302826, 1.4540570974349976, -0.48754024505615234, 0.1969224065542221, -1.419374942779541, 0.46260833740234375, -0.4475513696670532, -0.011614193208515644, -2.3394532203674316, 1.2927569150924683, 0.8664399981498718, -0.7022801637649536, -0.3082117438316345, -0.14114220440387726, -1.1186944246292114, -0.22826766967773438, -1.7676438093185425, 0.780475914478302, -1.854634404182434, 0.14585182070732117, -0.21793872117996216, -0.1714564859867096, -0.422478049993515, 0.311761736869812, 0.6812360286712646, -0.16874942183494568, 0.5010368227958679, 0.3698958158493042, -0.16703984141349792, -0.21906526386737823, 0.3290064334869385, -0.7930755615234375, 2.6165900230407715, 1.837124228477478, -1.4773412942886353, -0.18781818449497223, -0.48968884348869324, 0.21522068977355957, -0.1829582303762436, 0.799410343170166, 1.1311564445495605, -0.17486102879047394, -0.8194460272789001, -0.24719907343387604, -0.13565413653850555, 0.08096989244222641, 1.257858157157898, 1.2730952501296997, 0.4604148268699646, 0.42845776677131653, -0.7282954454421997, 0.43501657247543335, -0.5075852870941162, 1.363350749015808, 0.6240540742874146, -0.6385408043861389, 0.5295336842536926, -1.5737130641937256, 1.4337290525436401, -0.5095135569572449, 1.8715542554855347, -0.5878734588623047, -0.23209989070892334, -1.1618553400039673, -0.6420274376869202, -0.5819495916366577, 0.22686493396759033, -0.7681044340133667, -0.22338005900382996, 1.75911283493042, 0.29348504543304443, -0.3375494182109833, -0.1414562165737152, -0.44386208057403564, 0.7289140820503235, -1.9206057786941528, -0.6375303864479065, 1.6245383024215698, 0.7826833128929138, -1.4504663944244385, 0.8769005537033081, -1.311357855796814, -0.40252718329429626, -0.34897375106811523, 0.550107479095459, 1.6227290630340576, 1.4594916105270386, 1.735160231590271, 0.9913046360015869, 0.2136741727590561, -0.4581633508205414]], [[1.1076993942260742, 0.8583053350448608, 0.5794647336006165, -1.2527809143066406, 0.4026085138320923, -1.4229276180267334, -0.34636110067367554, -0.8430273532867432, -1.1597431898117065, -0.31605422496795654, 0.1725936084985733, -0.24671751260757446, 0.31625890731811523, 0.31217122077941895, -0.08145055174827576, -1.0588116645812988, 0.5710375308990479, -1.4930206537246704, 1.8739879131317139, 0.3502015173435211, 0.967481791973114, -1.864306926727295, -0.9908199310302734, -0.3173833191394806, -0.4914827346801758, 0.11077670753002167, -0.8387874364852905, -0.06360411643981934, 0.8949213027954102, -1.0380957126617432, -0.10767029225826263, 0.33655351400375366, 0.05171402916312218, -0.38284775614738464, 1.2987092733383179, 0.001431593089364469, -0.14322611689567566, 0.8039087057113647, 0.1936969757080078, 0.35961198806762695, -0.09420507401227951, 0.7098488807678223, 0.29397034645080566, 0.5647801160812378, 1.460439920425415, -0.24801355600357056, 0.4886167645454407, 1.0803977251052856, 0.012935410253703594, 0.34311357140541077, 0.05694521591067314, -0.6005028486251831, 1.0895110368728638, -0.2586357593536377, 0.3309182822704315, -1.7608200311660767, 0.7039235830307007, 1.1619423627853394, 1.0525376796722412, -0.7555282115936279, 0.9559136033058167, 2.482985496520996, -0.3782656490802765, -1.1103850603103638, 0.4675942659378052, 1.2729032039642334, 1.7814109325408936, -0.08629753440618515, -0.17889462411403656, -0.19687095284461975, 1.3218995332717896, -0.005649943836033344, 0.13404889404773712, 0.1537645012140274, -0.5571644902229309, -0.7188653349876404, -1.3691434860229492, -0.19029299914836884, 0.457477867603302, -1.4721108675003052, -0.11658726632595062, 0.18967095017433167, 1.6607062816619873, -1.4543794393539429, 1.128082513809204, 0.8287872076034546, -0.17315952479839325, 1.571698784828186, 0.11953137069940567, -1.0422992706298828, -0.8422266840934753, 0.6488940119743347, 0.3863900601863861, -0.8625608682632446, -1.6431195735931396, 0.4396791160106659, -1.3927757740020752, 0.1774720400571823, -0.9711137413978577, -0.2751311957836151, 2.262110948562622, -0.6924116015434265, -0.3691087067127228, 1.1186050176620483, -1.3074150085449219, -1.3600627183914185, -0.4354305565357208, -1.1552821397781372, 1.2696994543075562, -0.35174912214279175, 1.0417189598083496, 0.5261865258216858, -1.0025807619094849, -0.5526770949363708, -0.9061224460601807, 1.4113495349884033, 0.2095409631729126, 1.164105772972107, -1.2338529825210571, 0.5321823954582214, 0.7276721596717834, 0.8723159432411194, -0.6483586430549622, 1.0112613439559937, -1.2942003011703491, 1.0939456224441528, -1.207323431968689, 0.13563179969787598], [-0.16871052980422974, -0.05263996869325638, 0.19606655836105347, -0.6884415149688721, 1.3887404203414917, 0.8824211359024048, -0.6584615707397461, -1.4192322492599487, -0.23576951026916504, -0.6522965431213379, -0.52947598695755, -0.5643247365951538, -0.10163000226020813, 2.618453025817871, 0.7079459428787231, 0.08761315792798996, 1.1055020093917847, -1.546373724937439, 0.015748314559459686, 0.7232307195663452, -2.2886061668395996, -0.291574090719223, -0.7873558402061462, 0.9287768006324768, 1.2724225521087646, -1.9086936712265015, -1.1666501760482788, 0.08716557919979095, -1.0855196714401245, 1.1891192197799683, 0.10827796906232834, -0.6945876479148865, -0.8662428855895996, 0.11001288890838623, -0.9427385926246643, -0.8835549354553223, 0.5471633076667786, -0.6478891372680664, -0.8342620134353638, -1.5186536312103271, -1.4396313428878784, -0.6288567185401917, -0.5532518625259399, -1.38884699344635, -0.6062284111976624, -0.8177880048751831, -0.776456356048584, -0.04640901833772659, -1.5437599420547485, -0.7225281000137329, 2.5670485496520996, 0.265249639749527, -1.951173186302185, -0.9379822015762329, -0.3867643475532532, 1.1933985948562622, -0.12483252584934235, 0.7496703267097473, -1.093528151512146, 0.21629799902439117, 0.6418988108634949, 0.27120745182037354, -0.857613742351532, 0.39132869243621826, -0.14023075997829437, -1.3963128328323364, -1.1513725519180298, -1.496110439300537, -1.979812741279602, -0.16039013862609863, 0.5695792436599731, 0.1545809656381607, 0.026796307414770126, 0.6016359925270081, 0.12712682783603668, 1.6938890218734741, 0.5815937519073486, 0.23111674189567566, 0.37974482774734497, 1.0022047758102417, -0.27451378107070923, -0.6383793354034424, -1.4665759801864624, -1.900620460510254, 0.5906904339790344, -1.1884162425994873, 0.6480895280838013, -1.4581400156021118, 0.0459052175283432, -0.4196663200855255, 1.1221083402633667, -1.8563299179077148, 0.334891140460968, -0.08987421542406082, 1.2416640520095825, 0.6455533504486084, 1.0947332382202148, -1.294456124305725, -1.8701503276824951, -1.0957722663879395, -0.44113415479660034, 1.2224541902542114, 0.21021762490272522, 0.639354407787323, 0.9732160568237305, -0.06041767820715904, -0.6600289940834045, 0.6832697987556458, -0.13443897664546967, 0.04128090292215347, 0.8423104286193848, -0.8770370483398438, -0.4246244728565216, 0.42674708366394043, 0.8916111588478088, 0.7429800629615784, 0.29025813937187195, 0.7979369759559631, 0.42109015583992004, -1.147235631942749, -0.2560918629169464, -1.4243131875991821, 0.6189197301864624, 0.21531249582767487, -0.6670136451721191, 0.6594564914703369, -0.17144149541854858, 0.3183498680591583], [0.27435705065727234, -0.674622654914856, 1.1469262838363647, -1.3757970333099365, 1.4986374378204346, 0.20061491429805756, -0.6275359392166138, 1.058993935585022, -0.6242765188217163, -0.025376930832862854, -2.359663486480713, 0.32761892676353455, -0.4765835404396057, 0.1495988816022873, -0.4596062898635864, 1.50657057762146, -0.6562535762786865, 1.0928959846496582, -0.4996755123138428, -0.9893050789833069, -0.5876345634460449, -0.46419739723205566, 1.0976775884628296, -0.669692873954773, 0.06403074413537979, 1.1336658000946045, 1.5818537473678589, 0.4097191095352173, -0.462310254573822, -1.5024546384811401, -1.5227742195129395, -0.022032927721738815, 1.3586220741271973, -0.9458777904510498, 0.9835790395736694, 0.4496850073337555, 0.25104305148124695, 0.2239234745502472, -0.3708147704601288, -0.9441196918487549, 0.8708248734474182, 0.20590239763259888, -1.9102357625961304, -0.7126598358154297, -0.4799984395503998, -0.300680935382843, 0.6306824088096619, 0.9196546673774719, -0.6124337911605835, -0.3784550130367279, -1.0718361139297485, 1.4912135601043701, -2.2247395515441895, -0.8879178166389465, 0.1924930065870285, 1.3965909481048584, -0.8972735404968262, -1.342063069343567, -0.060398925095796585, 0.19984491169452667, 1.0034747123718262, -0.7205259799957275, -0.7689046263694763, 1.2083543539047241, -0.16390569508075714, -0.8223657011985779, 0.9703625440597534, 0.7893911004066467, -0.13096106052398682, 0.005046376492828131, 0.04057568684220314, 1.5302345752716064, -1.0066579580307007, -0.08354990929365158, -1.1114778518676758, 1.051993727684021, -2.682988405227661, 0.2276683747768402, 0.876185417175293, 1.6959680318832397, 1.7423900365829468, -1.592134952545166, 1.1748179197311401, -2.0041754245758057, -0.9050306081771851, 1.3055776357650757, -0.7023662328720093, 0.0540083684027195, 0.5082893967628479, 0.5645213723182678, 0.6715699434280396, -0.06965656578540802, -0.6045283079147339, -0.08457092940807343, 0.5963194370269775, -0.3368077576160431, 0.12876909971237183, 1.631101131439209, -0.3543594479560852, 1.5112532377243042, 0.04849785938858986, 0.45850783586502075, 0.6106276512145996, 2.1633126735687256, -0.49872711300849915, -0.9745104908943176, -1.3514702320098877, -1.1329596042633057, 0.016314052045345306, 0.9695044159889221, -0.8067443370819092, -0.3019534647464752, -1.5323338508605957, -0.2001555860042572, 1.5734198093414307, 0.8101712465286255, 0.22843898832798004, 1.0128740072250366, -1.5957655906677246, 0.11038421839475632, 0.7947611212730408, 1.7748924493789673, -0.7827701568603516, 0.17537030577659607, 0.1269301474094391, -0.47658610343933105, 0.8357906937599182, -0.47089624404907227], [-0.9675840139389038, -0.08897716552019119, -1.3022419214248657, 0.9818841814994812, 0.280497282743454, 0.3519893288612366, 0.7191445231437683, -0.2676721215248108, -0.2830095887184143, 0.966722846031189, -0.06099947541952133, -1.7579776048660278, -0.4805745482444763, -0.47622939944267273, -1.643537163734436, 0.13146139681339264, -0.5179879665374756, 1.4941827058792114, -0.49525773525238037, -1.4669374227523804, -1.042464256286621, 0.23168757557868958, -0.9824492931365967, 1.0296279191970825, 1.0713163614273071, -0.3739573657512665, -0.06279473006725311, -1.1898702383041382, -1.0959681272506714, 0.35101985931396484, 0.6691479086875916, 0.7057501077651978, 1.3705978393554688, -1.149857759475708, -0.9406343698501587, 0.6368207335472107, -0.9909575581550598, -0.29095518589019775, -0.567593514919281, 0.8321986794471741, -0.8013739585876465, -1.4123177528381348, 1.7477145195007324, -1.4449844360351562, -0.9072225093841553, 0.5947376489639282, -0.23945847153663635, -0.33351898193359375, 0.38313040137290955, -0.15621671080589294, 0.4974508285522461, -0.787260115146637, -1.198789358139038, 0.6003260016441345, 0.4120471477508545, -0.902747631072998, -0.7672590613365173, -1.7805390357971191, 0.1026936024427414, 0.14478212594985962, 0.14115774631500244, 0.9490054845809937, -0.23225103318691254, 0.3454034924507141, -0.24448339641094208, -1.7594877481460571, 0.9060531258583069, 1.4504152536392212, 0.635977029800415, -1.4806779623031616, 0.24019505083560944, -0.6603947281837463, -1.6381562948226929, 1.1668639183044434, 0.41285765171051025, -0.4736936688423157, 0.6913634538650513, 1.6402747631072998, 1.3783942461013794, 1.1534287929534912, 0.7264190912246704, 0.033378951251506805, -1.2540004253387451, 0.23607033491134644, 0.08559134602546692, -0.9147692322731018, 0.04048249498009682, 0.6822415590286255, 0.8762878179550171, 0.5934129357337952, -0.7410406470298767, -1.1464817523956299, 1.8509591817855835, -2.6738479137420654, -0.003839098149910569, -0.06418384611606598, -0.09561455994844437, 0.750111997127533, -0.6863630414009094, -0.1859898716211319, 1.4096800088882446, -0.4558658301830292, -0.23518186807632446, 1.3122096061706543, 0.03347688913345337, 0.451220840215683, 1.5867571830749512, 1.6716228723526, -0.9797325134277344, 2.6498355865478516, -1.148373007774353, -0.528171718120575, -0.4250430166721344, -0.9257892370223999, 1.5073602199554443, 0.08054304122924805, -0.7288140654563904, 0.3712471127510071, 0.49466732144355774, -0.977692723274231, 0.3912351131439209, 0.10498199611902237, -0.8693530559539795, -0.33206668496131897, 1.1635708808898926, 0.14249955117702484, -1.0005886554718018, 0.48554274439811707], [-0.5310352444648743, 0.9979732632637024, -1.2892332077026367, -0.8686363697052002, 0.8088476657867432, -1.1133651733398438, -0.18633508682250977, 1.7637425661087036, 0.8624670505523682, 0.35359328985214233, -0.2864733636379242, 0.5576390624046326, 0.926260769367218, -2.328258514404297, -1.986417531967163, 0.15113994479179382, 1.0743460655212402, 2.081512689590454, 1.3978166580200195, 0.6915210485458374, -0.567026674747467, 0.30672088265419006, -0.0025529242120683193, -2.2252871990203857, -0.26710236072540283, -0.211704820394516, -0.011092707514762878, 0.24449753761291504, 0.4412457346916199, -0.6755648255348206, 0.5144569873809814, 1.1328964233398438, -1.1943490505218506, -0.6589125990867615, 0.16434946656227112, 0.0990208238363266, 1.7876551151275635, 1.3358306884765625, 0.3220536410808563, -0.804440438747406, -0.2394615113735199, 1.2269902229309082, 0.6855579614639282, -0.3667376935482025, 0.269713819026947, -1.037360429763794, -2.4368114471435547, 1.1337668895721436, -0.9468812346458435, 0.3409383296966553, -1.685095191001892, 1.5111960172653198, 0.8347179889678955, -0.2802542448043823, -1.0591094493865967, 1.8711094856262207, 0.22909189760684967, -0.7099201083183289, -0.2975855767726898, -0.34320905804634094, 1.2966183423995972, 1.7382476329803467, -0.6136311888694763, 0.48982471227645874, 1.5630606412887573, -0.6603551506996155, 1.216564655303955, -0.40449368953704834, -0.4068441092967987, -0.3028734028339386, 1.5968092679977417, -1.2051793336868286, -0.8018410801887512, 1.2934997081756592, 0.3406120240688324, -1.2828806638717651, 0.4322868883609772, -1.116963267326355, -0.908605694770813, 0.897659957408905, 1.9784597158432007, 0.6502656936645508, 0.7288051247596741, 0.5759011507034302, -1.7311391830444336, 1.8363455533981323, -2.2424814701080322, 0.08257964253425598, -0.09411823004484177, -0.22054551541805267, 1.1541293859481812, 1.2990236282348633, 0.1292167454957962, 0.35310497879981995, -0.2745521664619446, 2.5269320011138916, -0.2585340738296509, -0.0602823942899704, -0.9122980833053589, 0.5229213237762451, -0.3019501566886902, -1.174950122833252, 0.17163726687431335, -0.5203589200973511, 0.584011971950531, -1.7569785118103027, 0.19267253577709198, -2.4441707134246826, 0.08659542351961136, 2.013617992401123, 0.019568998366594315, 0.39021217823028564, -2.251171588897705, 0.2533015012741089, -1.032015085220337, 0.48159360885620117, -0.5425240397453308, -1.0856103897094727, 0.7959150075912476, -0.3520989418029785, -0.8968695402145386, 0.478215754032135, 2.35646653175354, -0.816893994808197, 0.7553452849388123, 1.0480128526687622, 1.1913602352142334, -0.9571671485900879], [-0.599689781665802, 1.3798047304153442, 1.0431153774261475, -0.4431646466255188, -0.9655522704124451, 0.7846596240997314, -0.016204791143536568, 0.6596634387969971, 0.07602901011705399, -0.21313448250293732, -0.955129861831665, -0.6055287718772888, -1.474678874015808, 1.5125973224639893, -2.3835909366607666, 1.1106374263763428, -2.26185941696167, -0.7459801435470581, 0.09260619431734085, 1.0400160551071167, -2.025482177734375, 1.2738468647003174, 1.1104636192321777, 1.1620362997055054, 0.02363983355462551, -0.8001760244369507, -0.5064607858657837, -1.1569957733154297, 0.8233392834663391, 1.5341644287109375, -0.09057164937257767, 0.6058758497238159, -1.3932992219924927, -0.2843739688396454, 1.252837896347046, 0.14542636275291443, 0.4560084342956543, 1.0306400060653687, -0.0830092579126358, 0.7902342081069946, -0.05241258069872856, 0.7338742017745972, -0.004896307364106178, -1.657095193862915, 1.0159590244293213, -0.01484318170696497, 1.6259336471557617, -0.21150271594524384, 0.16772232949733734, -1.6684373617172241, -1.2178477048873901, -0.2201671153306961, -0.5014569163322449, 0.04378482326865196, 0.7523376941680908, -1.8757941722869873, -0.2757699191570282, 1.4032137393951416, -1.2667293548583984, -0.1609354019165039, -1.5133204460144043, 1.0679709911346436, 0.7909952998161316, -2.33058762550354, 0.7977433204650879, -0.16051623225212097, -0.7026624083518982, 0.35778096318244934, -1.1850601434707642, 0.09739809483289719, -0.35727402567863464, 0.0675051361322403, -0.4352230727672577, -1.142111897468567, -0.7694932222366333, 0.2650740444660187, -0.47904515266418457, 1.6190261840820312, 0.6003794074058533, 0.23002907633781433, 0.2755168080329895, 2.1277358531951904, 1.6318928003311157, -1.942173957824707, -0.6334441304206848, -0.4450646638870239, -1.5912244319915771, 2.2845122814178467, 0.8394858241081238, -0.2730543613433838, 0.12649081647396088, -0.023980669677257538, 1.4035062789916992, -0.2716987729072571, 0.6552269458770752, 0.08895666897296906, 0.932697594165802, 0.9078844785690308, -0.4888584017753601, 1.019994854927063, 0.7611916661262512, -0.978410005569458, 1.8972376585006714, -0.9471824765205383, 0.20516961812973022, 0.22375847399234772, 0.8392877578735352, -0.35484206676483154, -0.1592586785554886, -0.01976538635790348, 3.172086477279663, -0.594182014465332, -0.016767453402280807, -0.3529300391674042, 1.3014917373657227, 0.9274666905403137, -0.8434690237045288, -0.03210481256246567, 0.9443413019180298, 1.9276870489120483, -0.2640335261821747, -0.7431035041809082, 1.7767258882522583, 0.11132224649190903, 1.2576593160629272, 1.3889018297195435, 0.8313292264938354, -0.893389880657196], [-0.20753034949302673, 1.3389263153076172, -1.08504056930542, 0.7290350794792175, -1.5428524017333984, -0.5028529167175293, 1.5367766618728638, 0.6493490934371948, 1.3585686683654785, -0.5058830976486206, 0.540822446346283, -2.3810184001922607, -0.4233384132385254, 0.123955637216568, -1.7497174739837646, -0.9304232001304626, -0.6945977210998535, 0.07248622179031372, 0.8707059025764465, 0.10437610000371933, -0.450675904750824, -0.415985643863678, 0.4461859464645386, -0.28912994265556335, -1.0236332416534424, 1.6268011331558228, 0.07626799494028091, 0.5304749011993408, -1.66708505153656, -0.45940661430358887, -0.5907819867134094, 1.5400123596191406, 0.5591942667961121, -1.881746768951416, -1.0361731052398682, 0.21553358435630798, 0.724733293056488, 1.1500736474990845, -0.23479779064655304, -0.5072465538978577, 0.016554970294237137, -0.5710819959640503, -0.0603942796587944, 0.6929670572280884, -0.33041250705718994, -1.599756121635437, 0.8990767002105713, -0.7226125001907349, 0.19894547760486603, 1.4639716148376465, -0.40009385347366333, -3.3696701526641846, -1.6640287637710571, 1.7153708934783936, -1.0470317602157593, 0.5511164665222168, 0.0630926713347435, -0.5344865918159485, -1.6011461019515991, 1.0649724006652832, 0.41932278871536255, 1.4525774717330933, -0.09793190658092499, -1.2947689294815063, -0.44082003831863403, 0.03112366795539856, -1.9672273397445679, 0.4063100516796112, -3.296738862991333, 0.0010814964771270752, -0.5552958250045776, -0.02275128848850727, -0.5199277400970459, 0.2694055736064911, -0.5247126221656799, 0.14399954676628113, -1.1500974893569946, 1.0259242057800293, -1.0487112998962402, -0.6284606456756592, 1.029945969581604, 0.40584227442741394, -0.9732052683830261, 0.4114464521408081, -0.6049952507019043, -0.11315543949604034, -0.45874112844467163, -1.0584601163864136, -0.12471634149551392, 1.2139438390731812, 0.7959007024765015, 0.6989324688911438, 0.9174964427947998, -0.4547375440597534, -0.8476712703704834, 0.26136040687561035, 1.1461620330810547, -0.05952645093202591, 1.110465407371521, 0.9980179667472839, 0.7286797165870667, -2.579519748687744, 1.1479490995407104, 0.165994331240654, 1.1493678092956543, 0.5421838760375977, 0.25676921010017395, -0.4250393509864807, -0.1822415590286255, -0.8543760180473328, -1.4385173320770264, -1.2364109754562378, 0.9282928705215454, 0.6122984886169434, 0.001852346700616181, -1.91438889503479, 1.5046272277832031, -0.889115035533905, -0.08083825558423996, -0.36325323581695557, -0.8682292699813843, -0.4260410964488983, 1.3324365615844727, 0.7267586588859558, -0.5523912906646729, 1.4040497541427612, -0.008852191269397736, 0.09188316762447357], [0.005220571532845497, 0.09251866489648819, -0.5436851382255554, 0.05942964181303978, 0.07322545349597931, 0.22586873173713684, 0.06375952064990997, 0.030479177832603455, 1.4594132900238037, -0.7638933658599854, -0.3368484079837799, -0.870731770992279, -1.5392916202545166, 0.6690822839736938, 0.3324800431728363, 0.9009890556335449, -0.030613332986831665, -0.14325366914272308, 0.8321101665496826, 0.48928776383399963, 0.2144613265991211, -0.7943994998931885, -0.3283064067363739, -0.7025181651115417, -0.6191426515579224, -0.30332401394844055, 0.3243163228034973, -0.1770671159029007, -2.001534938812256, -1.0478333234786987, -0.5157373547554016, -0.6030033230781555, 0.899655818939209, -0.2792651057243347, 0.2748441696166992, 0.7189033031463623, 0.946098268032074, 1.5808336734771729, 0.7852492928504944, -0.08532781153917313, -0.4723912179470062, 0.7169989943504333, 1.3525010347366333, 1.4088318347930908, -1.067366361618042, -1.5431976318359375, -0.5313212275505066, 0.600570023059845, -2.483126640319824, 0.5103132724761963, 0.12501350045204163, 0.2534934878349304, 0.24505527317523956, -0.49104833602905273, 0.985643208026886, -2.3700969219207764, -0.5110558867454529, 1.1972864866256714, 1.583574652671814, 0.29181477427482605, 1.0833520889282227, 0.45432114601135254, -1.8214489221572876, -1.2173691987991333, 1.5731556415557861, 0.02091609500348568, -0.4066033959388733, 0.1945527344942093, -0.3336544334888458, 0.4130411148071289, -0.19741667807102203, -0.9384718537330627, -0.6099374294281006, 0.9550833106040955, 0.5488861799240112, 1.0834401845932007, 0.47929731011390686, -1.911191463470459, 0.9153371453285217, -1.9814419746398926, 0.40036115050315857, -0.6827436089515686, -1.3689218759536743, -0.971055269241333, 0.005914811976253986, 0.09192138910293579, -1.2494364976882935, 1.313126802444458, -0.1523282825946808, -0.0781637504696846, -0.16668741405010223, -0.48331907391548157, 0.26779988408088684, 0.43107128143310547, 0.37272313237190247, -0.7626054286956787, -0.5222906470298767, 0.645979106426239, 0.3848637640476227, -0.979439377784729, 1.2797621488571167, 0.019046910107135773, -0.4787876605987549, 0.10796403884887695, 0.6862567663192749, -1.6822106838226318, 1.4601898193359375, -0.5392717719078064, -1.6573940515518188, 0.9307287335395813, 0.4445725083351135, -0.35401833057403564, 0.2762930989265442, 2.447512149810791, -1.3840866088867188, 1.2117788791656494, -0.34822210669517517, 1.5533555746078491, -1.4013087749481201, 0.5410157442092896, -0.3824772536754608, -0.4724447727203369, -0.5016915202140808, 0.37516263127326965, 0.496506005525589, 1.53416907787323, -0.44442811608314514, 0.2259521633386612], [-0.40378081798553467, -1.321879267692566, -0.4839203655719757, -0.27066805958747864, -0.21354179084300995, -0.33454182744026184, 0.5718746185302734, -0.41237184405326843, -0.5870048999786377, -1.031467318534851, 2.579390287399292, -0.3274572789669037, -0.4974539875984192, -0.5486458539962769, 0.5441903471946716, 0.37001803517341614, 0.46260568499565125, 0.11716683208942413, 0.6782493591308594, -0.345333069562912, -1.3357957601547241, 0.32040947675704956, 0.09851615130901337, 0.36796027421951294, -0.1269402652978897, 1.6090457439422607, 0.5759109258651733, 0.16136564314365387, -0.7639476656913757, 0.5348368287086487, 0.34558233618736267, 0.21955399215221405, -0.4653799533843994, -0.5149383544921875, -0.5298518538475037, 1.0533742904663086, -0.5244532823562622, 0.39036744832992554, 0.2640021741390228, 1.0606355667114258, 0.14371609687805176, 0.002233466599136591, 2.3992300033569336, -0.3429439961910248, 0.14947736263275146, 0.539668083190918, 0.5330225825309753, 0.7252955436706543, 0.34465470910072327, -0.6661850214004517, -0.8933312892913818, 0.2183677852153778, -0.0455026850104332, 0.8152270913124084, 0.8112242817878723, 1.555546522140503, 1.9512611627578735, -1.2334403991699219, 0.2321041077375412, -2.401552200317383, 0.41839608550071716, 0.7358672022819519, 0.42953208088874817, -0.10350698977708817, -0.9658184051513672, -0.22585928440093994, -0.5665754675865173, 0.01653568260371685, -0.2286018580198288, -0.5494685769081116, -0.6647138595581055, 1.0083212852478027, -0.4964974820613861, 0.12974314391613007, -2.4701356887817383, 0.6066717505455017, -0.44799256324768066, 0.9406924843788147, 0.1391114741563797, 0.49773672223091125, 1.4468482732772827, -0.7458263039588928, 0.0524798259139061, 2.0068304538726807, -0.3445741534233093, 0.5036101937294006, -0.20026607811450958, -2.47100830078125, 0.7463608384132385, 0.06442989408969879, 0.28606316447257996, -0.9780065417289734, 1.359913945198059, 0.01002667099237442, 0.754296064376831, 0.9449195265769958, -0.6200803518295288, 2.063746929168701, -0.1431407481431961, 0.8364129066467285, -0.3514930307865143, 0.17838314175605774, -0.2086787074804306, -1.4292734861373901, 0.1288624107837677, -2.138017416000366, -0.9743074774742126, 0.013549069873988628, -0.8067446351051331, -0.4420322775840759, 0.7804325819015503, -0.9936470985412598, 0.13480976223945618, 0.8771423697471619, -0.7468798160552979, -0.6909682154655457, 0.5456886291503906, -0.5573576092720032, -1.4045825004577637, -0.12180273979902267, -0.4178282916545868, 1.1483001708984375, 0.2534772753715515, -1.4544548988342285, 0.062190763652324677, -0.593425452709198, 0.7720118165016174, 0.5116046071052551], [0.28888440132141113, -0.5866068601608276, 0.1602792739868164, 0.693302571773529, 0.521016538143158, 1.0647125244140625, -0.6504656076431274, 0.7405588626861572, 0.01369738020002842, 1.0819050073623657, -0.0336768738925457, -0.4189112186431885, -2.0867815017700195, 1.2745085954666138, -0.8144482374191284, 0.3558790683746338, -0.7954681515693665, 1.7780439853668213, 0.4641638994216919, -0.10222043842077255, 0.037120383232831955, -0.3377242386341095, 0.04018058255314827, 0.48605096340179443, -0.33293330669403076, 1.6774240732192993, 0.17378972470760345, -0.3077520728111267, 0.11374492943286896, 1.8696682453155518, 1.7679380178451538, -0.36198917031288147, -1.0221376419067383, -0.07135336101055145, 0.7188333868980408, 0.6210825443267822, 0.3803330659866333, 0.8881651163101196, -1.0150507688522339, -1.0456629991531372, 0.06251394003629684, 0.5236086845397949, 2.0914220809936523, -0.7084536552429199, 0.6899112462997437, 1.9736460447311401, 1.8385083675384521, -0.27834731340408325, -0.0054734437726438046, 1.6159186363220215, 1.4848504066467285, -0.5020974278450012, -1.6229082345962524, 0.07681829482316971, 0.113315150141716, -1.0793603658676147, 0.27090999484062195, 0.0670384019613266, 0.4519684910774231, 0.4793888330459595, 0.297956258058548, -1.8582813739776611, -0.08568426221609116, -1.7188071012496948, -0.43385830521583557, -0.03748931363224983, 1.1627752780914307, 1.3129329681396484, 0.6802713871002197, 0.34065449237823486, 0.5538301467895508, 2.4332096576690674, -0.16715772449970245, -1.5939831733703613, -0.39844921231269836, 0.07086916267871857, 0.07960195094347, 0.6306421756744385, 0.25133293867111206, -0.5911633968353271, -0.9953321814537048, 2.1440060138702393, -0.028811205178499222, -0.7553775310516357, -0.21377331018447876, -0.3678489625453949, -2.8105554580688477, 1.2573403120040894, -0.569837749004364, -0.1721109002828598, 1.2559741735458374, 0.7645497918128967, 1.8279637098312378, -0.4655538499355316, 0.5522035956382751, 0.03608859330415726, -0.7996222972869873, -0.08011220395565033, -0.49444249272346497, -1.951313853263855, 0.855506956577301, -0.5704406499862671, 0.09371039271354675, -0.034577980637550354, 1.4797568321228027, 0.38762980699539185, 0.4516918659210205, 0.29965972900390625, -1.008682131767273, -2.0775060653686523, -1.1310274600982666, 0.3338393568992615, 0.6213861107826233, 3.0439770221710205, -0.45098888874053955, 0.9330103397369385, 0.013357514515519142, 0.8366070985794067, -0.9488815665245056, -0.20375825464725494, 1.4076241254806519, -1.1985931396484375, -0.5579403638839722, -0.18701018393039703, 0.1768784373998642, 1.117898941040039, 0.7202273607254028, 0.9683425426483154], [-1.1936635971069336, -0.833726167678833, 0.5126834511756897, -0.23162759840488434, -0.9128232002258301, 0.0517575778067112, 0.3481510877609253, 0.31533610820770264, 0.2949220836162567, -1.6735808849334717, -0.12292385846376419, -1.4428118467330933, -0.5878299474716187, -0.5504500865936279, -0.5732855796813965, 0.04713593050837517, 0.8117555975914001, -0.7947601079940796, 0.9431254863739014, -2.5195095539093018, 0.18633154034614563, -0.44659411907196045, -0.619827926158905, -1.356999158859253, 1.236632227897644, 0.6224710941314697, 1.6809674501419067, 1.5513851642608643, 1.0727206468582153, 0.6850157380104065, -0.551917552947998, -0.23521311581134796, -0.21891792118549347, -0.12383167445659637, 1.2862296104431152, -1.5529425144195557, -0.10629135370254517, 0.009892215020954609, 0.2882971465587616, 0.5875000953674316, -1.1714441776275635, 1.4367656707763672, 0.7071740031242371, -0.5986575484275818, 1.522405743598938, 0.23183909058570862, -1.759665608406067, -0.15331122279167175, -0.022076869383454323, -0.7907960414886475, -1.6420835256576538, 0.07085511088371277, 0.40499112010002136, -1.1655868291854858, 1.8386787176132202, -1.2918776273727417, 0.36791279911994934, 2.7386186122894287, 2.26711368560791, 0.4448696970939636, 0.1700972616672516, 1.105394721031189, 1.9757732152938843, 0.1936054229736328, -0.35521402955055237, -0.666450023651123, -0.41972070932388306, -2.3592214584350586, -0.9314976930618286, 0.5431875586509705, -1.8222142457962036, 2.0847866535186768, -0.0432993620634079, 1.1229579448699951, 0.08427304029464722, 1.4630184173583984, 1.2119014263153076, -0.5940777063369751, -0.020009750500321388, 0.7860419750213623, -0.9556103348731995, -1.3153131008148193, -0.01357786264270544, 0.8676455616950989, 1.3778706789016724, -0.07441964745521545, -0.28437137603759766, -0.9929860830307007, -0.4559212625026703, -0.9598575830459595, 1.2694119215011597, -0.4027559161186218, -0.22357456386089325, -0.0615864098072052, 1.1972452402114868, -0.36259135603904724, 0.48203712701797485, 0.8367885947227478, 1.319070816040039, -0.9040067195892334, -0.1063695177435875, 1.3853123188018799, 1.740691900253296, 0.3841612935066223, -1.7090425491333008, 0.6775802373886108, -0.04339433088898659, -1.1474226713180542, 0.6294137239456177, -0.7472178936004639, 0.02104754000902176, 0.06424914300441742, -0.5357578992843628, -0.5960375666618347, -1.831666350364685, 0.8074151873588562, 0.9836308360099792, -1.9040626287460327, 1.0731513500213623, 0.773410439491272, -0.04375180974602699, -1.5848631858825684, 0.2554419934749603, 0.8087337017059326, 0.690667450428009, -0.704988956451416, 0.4730563163757324, 0.4170759320259094], [1.2875531911849976, -1.168192982673645, -0.12281013280153275, 0.20120352506637573, 0.6767215728759766, -0.9880077838897705, -1.9047917127609253, 0.3951982259750366, -0.05864809826016426, -0.0026411779690533876, -1.0884935855865479, -0.3413265645503998, -0.6273413300514221, -0.48105770349502563, 0.15416277945041656, 0.5855080485343933, -1.4308137893676758, 1.4856324195861816, 0.11423175781965256, 0.07402516901493073, -0.7805881500244141, -0.6158117651939392, 0.5488871335983276, -0.6112062335014343, 2.4272727966308594, -0.7647629380226135, 1.750801682472229, -0.46835142374038696, 1.3235467672348022, 0.5979113578796387, 0.7624006867408752, 0.3831263482570648, 0.7157405018806458, 0.41520822048187256, 1.2048581838607788, -0.11310611665248871, -0.5118644833564758, 0.21501965820789337, 0.38847002387046814, 1.3803575038909912, -0.23374925553798676, 0.20087288320064545, -0.4927847683429718, -1.4892463684082031, -0.7611749172210693, 0.8899253010749817, -0.5551276206970215, -2.0298879146575928, 0.3847900927066803, 0.261355459690094, 3.628056287765503, 0.3093078136444092, 0.08859843015670776, -1.6394188404083252, 0.32490870356559753, 1.3012099266052246, 1.975040316581726, -1.1788561344146729, -1.1361697912216187, 0.5002658367156982, 0.1497815102338791, 0.4604388177394867, -0.501716136932373, 1.0070879459381104, -1.1685731410980225, -0.1982261687517166, 0.49219968914985657, -1.5551599264144897, 0.9367474317550659, 0.5967350602149963, -0.09872148185968399, 1.4417330026626587, 0.3329324722290039, 0.23193170130252838, 1.3250869512557983, -0.7738732695579529, 0.011970223858952522, 0.5066462159156799, 0.737525224685669, -1.9094130992889404, -0.823104977607727, -2.1160411834716797, 1.8885881900787354, 0.47320133447647095, -0.6771088242530823, -0.8326218128204346, -0.057455938309431076, 0.6051860451698303, 0.02236137166619301, 0.3016325831413269, 1.7828385829925537, 0.634060263633728, -1.9089335203170776, 0.7435222864151001, 2.1557443141937256, -0.0822470560669899, -2.620546579360962, 0.9743567109107971, 0.33018508553504944, 0.9725599884986877, 1.4513797760009766, -2.029459238052368, 1.3212130069732666, 1.0491857528686523, -0.8616384267807007, 1.0944205522537231, -2.03867506980896, 2.7913119792938232, -1.217539668083191, -0.8857897520065308, -0.956088125705719, -0.12858133018016815, 0.38283464312553406, -0.9676647186279297, -0.6708357334136963, -1.9476803541183472, -0.3789007365703583, 0.46867185831069946, 0.9396589994430542, 1.1047933101654053, 1.3135377168655396, -0.5915839672088623, -1.2040473222732544, -0.017088057473301888, -2.0681443214416504, -1.1636505126953125, 0.43948373198509216, 1.4888689517974854], [-0.3363973796367645, -0.26977160573005676, 1.3177566528320312, -1.5800998210906982, -0.37008675932884216, 0.7442012429237366, -1.0299413204193115, 0.050946757197380066, 1.233012080192566, 0.35576701164245605, -0.626954197883606, 0.7965498566627502, -0.3992697298526764, -0.9338247776031494, -0.9054528474807739, -0.8746749758720398, 0.7258586287498474, 0.05774332210421562, -0.21722900867462158, 0.9315308928489685, 0.3135005831718445, 0.10087458789348602, 0.5619313716888428, -0.2821836769580841, 0.19161255657672882, -1.1563810110092163, -1.8828096389770508, -1.5259101390838623, -1.5839781761169434, 0.4163469970226288, -0.30216193199157715, 1.4194263219833374, 0.4717828035354614, 1.644407033920288, -0.8909318447113037, 0.44969525933265686, 0.9978461265563965, -1.237444281578064, -2.4371554851531982, 0.12425092607736588, 2.1562302112579346, 2.4792566299438477, -0.5942758321762085, -0.4114724397659302, 0.8287816047668457, -1.5510821342468262, 0.6247301697731018, -0.8070228695869446, -0.8610561490058899, -0.3247166574001312, -0.24841105937957764, -0.9532990455627441, -0.5872573256492615, -0.994132399559021, 1.085358738899231, -2.0512917041778564, 0.15460239350795746, -1.1754943132400513, -1.5895521640777588, -0.6169967651367188, 1.3605018854141235, 0.2626071572303772, 0.1613936722278595, 0.1408078372478485, -0.13656528294086456, 0.7179415225982666, 0.9585882425308228, 0.08610918372869492, 0.6934661865234375, -0.5810402631759644, 0.9484363198280334, 1.225196361541748, 0.6604206562042236, 1.3226113319396973, 1.1295349597930908, 0.7680504322052002, 0.18587522208690643, -0.032747533172369, 1.0432405471801758, 0.2655785083770752, 0.719266951084137, -0.23943829536437988, 0.08771464228630066, 1.2167471647262573, 0.3024413287639618, 0.193434476852417, -0.1314043551683426, -1.9748862981796265, 1.0529991388320923, 0.053702179342508316, 0.20985101163387299, -0.1273883730173111, 0.4254468083381653, -1.3974577188491821, -0.07225897163152695, -0.06523142009973526, -0.5409975051879883, -0.7646211385726929, -0.1797686070203781, -1.118353247642517, -0.15040193498134613, 2.8175792694091797, 0.62641841173172, -0.2970343232154846, -0.5948350429534912, -0.508906364440918, -2.356708526611328, -0.47260770201683044, -0.463515967130661, -1.3129973411560059, 0.7595486640930176, -0.4286718964576721, -1.2148617506027222, -1.3490960597991943, 0.6155203580856323, -0.8637431859970093, 0.4081306457519531, 0.9335798025131226, 0.7447091341018677, -0.5183354020118713, -0.10761440545320511, 0.6749663949012756, 1.2863590717315674, -0.4680078625679016, -0.08828659355640411, 0.2946632206439972, -0.9615539908409119, -1.219478726387024], [1.5917856693267822, 0.7284156680107117, -0.367793470621109, 0.7416527271270752, 0.2565848231315613, 0.8174880146980286, 0.1366463005542755, 0.30048057436943054, -1.0266451835632324, -3.1607768535614014, -1.129581093788147, -0.06788461655378342, -0.19252772629261017, -0.5317206382751465, -0.26343002915382385, 0.1282954216003418, -0.16582992672920227, -1.1131415367126465, 1.41361665725708, -0.5824548006057739, 0.6529858112335205, -0.336608350276947, -1.0920076370239258, -0.3889177739620209, -1.3359507322311401, -0.7205477356910706, -1.846193790435791, 2.2786734104156494, 0.07620322704315186, 0.28335291147232056, -0.7627835273742676, -1.5475248098373413, -0.13952194154262543, -0.458202064037323, 1.261029839515686, 1.7764006853103638, 0.07059625536203384, -0.5367347598075867, -0.6415704488754272, -0.058359790593385696, -1.1330533027648926, 1.2173763513565063, 0.0037436154671013355, 0.597237229347229, 0.11829816550016403, -1.1801363229751587, 0.6220168471336365, 0.39458200335502625, 1.3597279787063599, 0.30935755372047424, 0.5353837013244629, 0.05058586969971657, -0.1577034443616867, 0.13983453810214996, 0.20205968618392944, 1.207863688468933, 2.584200143814087, -0.7676609754562378, -0.7827709913253784, 2.0455198287963867, -0.22293724119663239, 0.5949972867965698, -0.4387100338935852, -0.1726098358631134, -0.0735311508178711, 0.05523717775940895, 1.8071790933609009, 0.6152368783950806, -0.5095621347427368, 0.5773578882217407, 0.20673038065433502, -0.6471989154815674, 0.12283367663621902, 0.18213146924972534, 0.3361331522464752, 0.33502012491226196, -0.17632165551185608, -0.9909677505493164, 0.075745590031147, 0.8275831937789917, 0.2876111567020416, -2.141190767288208, 0.5739312171936035, -0.8958147764205933, 0.6312036514282227, -0.2669818103313446, -0.37201234698295593, -1.7868797779083252, 0.9944814443588257, -0.7960442900657654, -0.20505556464195251, 0.37802842259407043, -0.5561161637306213, -1.7825968265533447, -0.2921326756477356, 0.5790577530860901, 0.983555257320404, -0.939784586429596, 1.5461660623550415, -0.13970215618610382, -0.7805594205856323, 0.9241055250167847, -0.7166461944580078, -1.3614784479141235, -1.594089150428772, 1.0275053977966309, -2.4914424419403076, -2.042206048965454, 0.9472874999046326, 1.5892177820205688, 0.5436229705810547, 0.046265218406915665, 0.751183032989502, 0.25759461522102356, -0.9298903942108154, -0.783897340297699, -0.27534720301628113, 2.2242510318756104, 1.9093660116195679, -0.21457800269126892, -1.1014443635940552, 0.49189838767051697, -1.1925235986709595, 0.3544575870037079, -0.159718856215477, 0.009680412709712982, -0.12871746718883514, -1.187351942062378], [-1.3328622579574585, -1.2455179691314697, 0.6693905591964722, 0.07957448810338974, -0.4475753605365753, -1.226396918296814, 1.3485968112945557, -0.1840602606534958, 0.8857626914978027, -0.5693506598472595, 1.2052717208862305, -1.0420018434524536, -1.4984668493270874, 0.5233270525932312, 0.053494155406951904, 0.10019967705011368, 0.23374336957931519, -0.1946631669998169, -0.39921826124191284, 0.03741887956857681, -0.6685444116592407, -2.7407233715057373, 0.6627411842346191, -0.37528327107429504, -0.5697662234306335, 0.4079528748989105, 1.423917293548584, 0.0372125543653965, 0.26906076073646545, -0.003397422842681408, -0.2579014301300049, -1.3749420642852783, 0.5970349907875061, 0.4336403012275696, -0.4023255705833435, 0.3684830963611603, 0.4140196144580841, 0.7723495960235596, -0.6962589025497437, -0.7403298020362854, 1.2665668725967407, -0.3404725193977356, 0.8278546333312988, -1.1252448558807373, 0.32981735467910767, -0.8294464945793152, 0.9126453399658203, 2.221510648727417, 0.47038233280181885, 1.815760850906372, -0.7545296549797058, 0.7574813365936279, -0.9284238219261169, -0.22860993444919586, 0.4506261646747589, -0.7126430869102478, -0.9631595015525818, -1.8807923793792725, 0.12035314738750458, -1.0913654565811157, -1.225767970085144, 0.36671602725982666, -0.5712437033653259, 0.034992508590221405, 0.42339298129081726, -1.5699408054351807, -1.249396800994873, -0.6682298183441162, -0.6109892725944519, -0.1199907660484314, -0.8524981737136841, -1.4390147924423218, 0.2321161925792694, -0.36019688844680786, 0.7027862668037415, -0.003476398531347513, -0.6572201251983643, 0.6342712640762329, 0.32068660855293274, 1.523894190788269, 0.11135829240083694, -1.0344878435134888, 0.9169995188713074, 0.09167113900184631, -0.5630279183387756, 2.5764572620391846, 1.5974407196044922, -0.05807389318943024, -1.541632056236267, 1.1668418645858765, 0.40412279963493347, -0.629951536655426, 0.8920204639434814, -0.3178703784942627, -0.33884096145629883, -0.24307945370674133, -0.2107595056295395, 0.10519540309906006, 1.0620169639587402, 0.4757576584815979, 0.8713993430137634, 0.028704650700092316, 0.041377995163202286, 0.43280014395713806, -1.636413335800171, -0.022786181420087814, 0.763875424861908, -0.639141857624054, 1.4524635076522827, -1.0926765203475952, 0.7878145575523376, -0.07311831414699554, -1.074854850769043, 0.09429693222045898, -0.3974568545818329, -0.11375638842582703, 0.2002890408039093, -1.3679478168487549, 2.4432311058044434, 1.2811877727508545, -0.8899930119514465, 0.7325131297111511, 1.7625590562820435, -0.878197431564331, 1.0326831340789795, 0.9175485372543335, 0.199118971824646, -0.08842679113149643], [-2.6822292804718018, -1.283219814300537, 0.5880749821662903, 1.0586618185043335, -1.2739524841308594, -0.1310848891735077, -0.9686300754547119, -0.36353859305381775, -0.3524061143398285, -1.7549529075622559, -0.01633806899189949, 0.3387318551540375, 0.46264177560806274, 1.049930453300476, 0.46634912490844727, 0.3792836368083954, 0.40515825152397156, 0.05337929353117943, 0.5740495324134827, 1.0710257291793823, -0.35559937357902527, -0.46885624527931213, -0.8504054546356201, 0.8747239112854004, 0.035785235464572906, 0.2297043800354004, 0.21878570318222046, 0.43262049555778503, -1.758481740951538, 0.5618868470191956, -0.8024112582206726, -0.5361002087593079, 0.5160380005836487, 0.3995437026023865, 1.7563560009002686, -1.042222499847412, -0.5008075833320618, 0.5150162577629089, -0.5317471623420715, 0.553753674030304, -0.32973477244377136, -0.5177100896835327, -1.9194414615631104, 0.860209047794342, 0.9658453464508057, 1.7114317417144775, 0.5830729007720947, -1.2117723226547241, 0.17526015639305115, 0.6778084635734558, -0.3438817858695984, 0.22533218562602997, 1.9702767133712769, 0.5074809789657593, -0.37618929147720337, -0.35581186413764954, 0.07921086996793747, -0.8482910394668579, 0.9689446687698364, 0.5762333273887634, 0.21960264444351196, -0.7699545621871948, -0.03629766404628754, 1.0028834342956543, -1.7210620641708374, -0.3451308012008667, 0.37892961502075195, 0.5406364798545837, -1.3514328002929688, -0.19527018070220947, 0.7319114208221436, -1.1134682893753052, 0.8206345438957214, -0.7783151865005493, -0.06456564366817474, -0.996026873588562, 0.6306498646736145, 2.4295506477355957, -0.1806716024875641, 0.035463061183691025, -0.18483878672122955, 0.6609532237052917, -0.7715031504631042, -0.4388806223869324, 1.8797857761383057, 0.5177565813064575, -0.7829561233520508, -0.9449455738067627, -1.0979657173156738, -0.38303542137145996, -0.21206626296043396, 0.025886405259370804, 1.065293312072754, -0.9708934426307678, -0.9250289797782898, 0.6082362532615662, -0.16002660989761353, 0.72447270154953, -0.08731041848659515, -0.6808103322982788, 0.08062275499105453, -1.00186026096344, -1.0192344188690186, -1.0767626762390137, 0.1081453487277031, -1.5444563627243042, -1.5893384218215942, -0.1260843724012375, -0.5246180891990662, 2.40354585647583, -1.900143027305603, 0.8965290188789368, 0.08607923239469528, 0.441391259431839, -1.9267468452453613, -0.016268812119960785, 0.10622826218605042, -2.4716014862060547, 0.2525452673435211, 1.445751428604126, -0.7631171941757202, 0.8140339851379395, 1.8312969207763672, 0.42757648229599, -0.9037501811981201, -0.7785807251930237, 1.1204419136047363, -0.8165149688720703], [0.8636767268180847, -0.7885255217552185, 0.21703435480594635, 1.2691751718521118, -0.20160016417503357, -0.9217184782028198, -0.5797685980796814, 0.09455705434083939, 1.7155160903930664, -1.6741375923156738, -1.3036613464355469, -0.08671282976865768, 0.32821154594421387, 0.9968942403793335, -0.07977791130542755, -1.475748896598816, 2.1378061771392822, -0.47835907340049744, -1.8893922567367554, -0.9873400330543518, -2.236645460128784, -0.6228145360946655, -0.35295361280441284, -1.7602767944335938, -1.7943693399429321, 0.554712176322937, -0.4936012327671051, -0.4398025572299957, 1.4092292785644531, -0.6520885825157166, 0.5842642784118652, 1.3582584857940674, -0.6901649236679077, -0.8655229806900024, -0.2171509861946106, -1.0523453950881958, 0.10848021507263184, 1.0950766801834106, -0.26721957325935364, -1.6026325225830078, -2.724442481994629, -1.0025466680526733, -0.16062100231647491, 0.6766291260719299, -1.1173970699310303, -0.2211543470621109, -1.0770814418792725, -0.5555038452148438, -0.45909541845321655, -0.267214298248291, 0.08101115375757217, -0.44283074140548706, 0.6406290531158447, 0.5281645059585571, 0.2926884591579437, -0.8423601984977722, 1.2671211957931519, -0.17153462767601013, 0.06463447213172913, 1.3834478855133057, -0.009755118750035763, -0.7994958162307739, 1.2270878553390503, 0.14617487788200378, 0.18753761053085327, 0.6664932370185852, -0.28041908144950867, 1.3053690195083618, -0.9134758114814758, 1.9215706586837769, -1.4509080648422241, -1.7159818410873413, 0.5346906781196594, 1.2936478853225708, -1.0955408811569214, 1.679485559463501, 2.260903835296631, 0.7154737710952759, -0.5154518485069275, -0.5724881291389465, 0.7138680219650269, 0.7221677303314209, 1.026999592781067, -1.5520780086517334, -0.028193065896630287, -2.1705965995788574, -0.5243986248970032, 1.6465719938278198, -0.15058943629264832, 0.535528302192688, -0.2165827453136444, -1.745382308959961, -1.148349404335022, -1.2519099712371826, -0.09046326577663422, 0.4586709141731262, -0.073271244764328, 0.085360586643219, 0.28088459372520447, -0.39861786365509033, -0.6436163187026978, -1.5355784893035889, 0.04068649560213089, 0.6734519600868225, -0.36814191937446594, -0.623533308506012, 0.9987263679504395, -0.5231866240501404, 0.6835430860519409, 0.9195486307144165, -0.06030784174799919, -1.8295649290084839, 0.6537225842475891, -1.6979341506958008, -1.8133388757705688, 0.15401394665241241, 0.17931964993476868, -0.45600613951683044, -0.5809674859046936, 1.349733591079712, -0.4440072178840637, -0.182834655046463, -0.03735658526420593, -0.9557289481163025, 1.8126766681671143, -0.483023464679718, -1.090812087059021, -0.24723325669765472], [1.1628819704055786, -2.884859800338745, 1.1767946481704712, -1.811696171760559, 0.7686130404472351, -1.053409218788147, -0.712454080581665, -0.6836146116256714, 0.07806327193975449, -0.3136974275112152, 0.4833444058895111, -0.8057482242584229, -0.9396834373474121, -0.9590671062469482, -1.4345132112503052, 1.962743878364563, -1.4269731044769287, -0.38042521476745605, -0.2139734923839569, -0.5946629047393799, -1.2476770877838135, 0.6921463012695312, 0.6407129764556885, 0.08697815984487534, 0.14085394144058228, 2.5160396099090576, -0.8925973176956177, -0.8768798112869263, -1.8325157165527344, 1.7314984798431396, -0.17630410194396973, 1.595010757446289, 0.49686339497566223, -0.747276246547699, 0.9298828840255737, 0.5753633379936218, 0.07927396893501282, -0.0669383704662323, 1.3967387676239014, 1.7617218494415283, 0.6031410694122314, 0.2460787445306778, 0.3393939733505249, -0.27040407061576843, 1.500708818435669, 1.0366289615631104, -0.18214210867881775, -0.1584901064634323, 0.13530834019184113, 0.23770248889923096, 0.2475871443748474, 0.8839501738548279, 1.344763159751892, 0.810399055480957, 0.3636815547943115, 0.7183451056480408, 0.6245711445808411, -0.2209070324897766, 0.387658029794693, -0.9475402235984802, -0.033290423452854156, 0.42325374484062195, 0.023724982514977455, -0.7131056785583496, 0.9798415303230286, -0.3007260262966156, 0.19150753319263458, 0.1360742300748825, 1.7799334526062012, 0.20176967978477478, 0.8263869881629944, -0.07800287008285522, 0.8790539503097534, -0.8149766325950623, -1.846347451210022, 0.6475018858909607, 1.1273326873779297, -0.24920061230659485, 0.2638370096683502, -0.2302238494157791, 0.3547617197036743, -0.8780379295349121, -0.2811872959136963, 1.4763422012329102, -1.2644551992416382, 0.6770315766334534, -1.0975978374481201, 0.3183385133743286, -1.3845099210739136, -0.6112096309661865, -0.6438496112823486, -2.4846384525299072, 1.363906979560852, 1.1704436540603638, -2.158076286315918, -0.4513823390007019, 0.8788386583328247, -0.5690927505493164, -0.3463752269744873, -1.7794109582901, 0.0752711221575737, 0.607671856880188, -1.7371516227722168, -0.30347487330436707, 0.31614020466804504, 0.7089557647705078, 0.3721287250518799, -1.161034107208252, 0.7221329212188721, 1.4473786354064941, -0.22835582494735718, -2.1199028491973877, -2.289767026901245, 2.4856698513031006, -2.2516322135925293, 1.3971327543258667, -0.5285748839378357, -0.4325672388076782, -0.38116025924682617, -1.188938021659851, 1.2719275951385498, 0.22310569882392883, -1.2438791990280151, 0.15701881051063538, -0.05048763006925583, -0.9612306952476501, -0.21662671864032745, -1.6555876731872559], [1.0838459730148315, 0.8534111976623535, 1.555675745010376, -0.5625017285346985, -0.6598206758499146, -0.5291804671287537, 0.7032952308654785, -0.3624475598335266, 0.014856467954814434, -0.6367495059967041, 1.3103508949279785, -0.1782844364643097, -0.6188283562660217, 0.28496357798576355, -0.36763623356819153, -0.20305661857128143, 1.3823577165603638, -1.7348304986953735, -0.032979294657707214, 0.016786394640803337, 1.3373280763626099, -0.5757356882095337, 0.7983928918838501, 0.6889593005180359, 0.24770238995552063, 0.24974411725997925, -0.5630154609680176, -0.6213963627815247, 0.6635025143623352, 0.9912076592445374, -0.362537145614624, 0.6492491960525513, 0.40415847301483154, 0.5252935290336609, 0.09751324355602264, -0.7034393548965454, -1.5084952116012573, 0.9680293202400208, -1.9340331554412842, 0.941637396812439, -0.8748734593391418, -0.17733371257781982, 0.025644641369581223, 0.2913360595703125, -0.665499746799469, 0.5664812922477722, 0.738105058670044, 0.8218844532966614, 1.3961617946624756, -1.9432706832885742, -0.7548326849937439, 0.9601895809173584, 0.024167895317077637, 1.681728720664978, 2.0019545555114746, 0.38022348284721375, -1.514849305152893, -0.816076397895813, 0.9495342969894409, -0.4994799494743347, -1.0250988006591797, 0.6379600167274475, 0.8297111392021179, 0.4647338390350342, 0.6232131123542786, -0.46635690331459045, 1.9654567241668701, -0.18294809758663177, -2.357433795928955, -0.8849310278892517, 0.2651071548461914, -1.7087249755859375, -0.7086053490638733, 0.429943323135376, -1.0171595811843872, 0.5680363178253174, 1.150056004524231, -0.7177767753601074, -0.8907848000526428, -0.5360509157180786, -0.11781682074069977, -0.6517043709754944, 0.6071876287460327, 1.2042224407196045, -0.56056147813797, 1.3975646495819092, -1.1460270881652832, -1.0445832014083862, -0.5451844930648804, 0.5946352481842041, -1.7588483095169067, -0.4152529537677765, -1.1460788249969482, -0.3479855954647064, 0.5195619463920593, -0.860106348991394, 0.5796586275100708, 0.9483630061149597, 0.5801347494125366, 0.0899987742304802, -0.46781063079833984, -1.311052680015564, 1.1853817701339722, -0.9957860112190247, 0.46887949109077454, -0.047160662710666656, 0.708524227142334, 0.3772416412830353, -0.3899801969528198, -0.35149645805358887, 0.4644987881183624, 2.612544059753418, 0.1309128701686859, -0.6049224138259888, 0.5801020860671997, -1.0264501571655273, 1.6341073513031006, -1.0705527067184448, 0.002197175519540906, -1.5510796308517456, -0.24199287593364716, 0.9758552312850952, 1.0667189359664917, -0.2586480677127838, 2.439218759536743, -2.4052541255950928, -0.8201248645782471, -0.565889835357666], [0.16284134984016418, 1.269098162651062, 0.38108184933662415, -0.08704628050327301, -0.3433389961719513, -0.8809096813201904, -0.7175902724266052, -1.1011964082717896, 1.7424237728118896, -1.6367453336715698, 1.6130050420761108, -0.795543372631073, 0.3230423629283905, 0.6352686882019043, 2.6040375232696533, 0.12254011631011963, 0.08067192882299423, -0.5304260849952698, -1.4318478107452393, -0.6419383883476257, 0.8085657358169556, -0.017502903938293457, 0.9536073207855225, 0.15184910595417023, 0.5316066145896912, -2.586042642593384, -1.367991328239441, -1.056514859199524, 0.3833232820034027, 0.8053098320960999, -1.4470405578613281, 1.1051867008209229, -0.1208539754152298, 1.5699976682662964, -0.14294202625751495, -0.6362648010253906, -1.2582099437713623, -0.22465211153030396, -0.20023353397846222, 0.30581608414649963, -0.8348308205604553, 1.2516714334487915, -0.3964468836784363, -0.40020498633384705, 0.5803450345993042, -0.004310987889766693, 1.144355058670044, 2.6654105186462402, -0.16759298741817474, 0.5150274634361267, 1.744295597076416, 0.27878597378730774, -0.978243350982666, 0.7247764468193054, 0.987295389175415, 0.3233948349952698, 0.7600394487380981, 0.43635237216949463, -1.2625393867492676, -0.08349768072366714, 1.5251712799072266, -0.5658084154129028, 0.08036836236715317, -0.49716857075691223, -1.1007157564163208, -0.7441757917404175, -0.8847998380661011, -0.8770734667778015, -1.5221210718154907, -0.11826969683170319, -0.647783100605011, -1.8605737686157227, -0.11612295359373093, -1.802621603012085, -2.175126314163208, 0.034841470420360565, 1.4608173370361328, 1.9537709951400757, -2.061802625656128, -0.4744875133037567, 0.005088237579911947, 1.0673962831497192, 0.7482571601867676, 0.8389962911605835, 1.703251838684082, 1.105377197265625, 0.8837214708328247, -0.6220988631248474, -1.311279535293579, -0.03932754695415497, 0.44298848509788513, 0.33326148986816406, -0.47931450605392456, -0.7392563223838806, 1.8647668361663818, 0.5689531564712524, -1.4813776016235352, -1.0129899978637695, -0.5227867364883423, -1.047481894493103, 0.7358501553535461, -1.1043514013290405, -1.4084606170654297, -0.6290997266769409, -0.48785099387168884, -0.4771713316440582, -1.2253642082214355, -1.004721760749817, -1.5416405200958252, 0.9589874744415283, 0.4596441686153412, 0.7455124258995056, -0.05335557088255882, 0.4593876898288727, -0.8477275371551514, 1.4193464517593384, -0.6114919185638428, -0.4773297905921936, 0.13468009233474731, -0.8851759433746338, -1.5822409391403198, -2.9639437198638916, -0.14402200281620026, 0.51697838306427, 0.5301941633224487, -1.8771135807037354, 1.6755249500274658, -0.05860744044184685], [-0.5847204327583313, -0.4723355174064636, 0.7984519600868225, 0.8930484652519226, 1.161597728729248, 0.8602150082588196, -1.3502591848373413, 1.3449287414550781, -1.5469707250595093, 0.9886939525604248, -0.4237498939037323, 1.3238641023635864, 1.6565347909927368, 1.453568935394287, 1.4401905536651611, -0.4091184735298157, -0.08609944581985474, -0.5214450359344482, -2.8479342460632324, 0.22362133860588074, 0.06321784108877182, 0.2735210359096527, 1.0775256156921387, -0.20105525851249695, -0.6350007653236389, -0.6053737998008728, -1.7125587463378906, 1.3419129848480225, -0.8903622031211853, -0.5052846074104309, 0.6231683492660522, 0.09306970238685608, 0.19585967063903809, -1.6591275930404663, 0.02245546318590641, -0.29001563787460327, -0.590531587600708, 0.829010546207428, 0.35430410504341125, -0.20244364440441132, -0.8454909920692444, -0.49260053038597107, 0.6465104818344116, -0.33777716755867004, -0.6765355467796326, 1.4917829036712646, 1.324765920639038, 0.5392838716506958, 0.6704947352409363, 0.4168548583984375, -0.11648687720298767, 2.544917345046997, 0.8183068037033081, -0.14418506622314453, -1.5787501335144043, -0.5712451338768005, -1.1841450929641724, -0.8480871319770813, -1.5016177892684937, 0.4540044367313385, -0.8289206027984619, -0.5374387502670288, -0.6251757740974426, -1.3939199447631836, -0.8799010515213013, 0.5888693928718567, -0.023564159870147705, -0.1533026546239853, 1.0658106803894043, -0.6090770959854126, 0.8004617691040039, -0.2362890988588333, -0.7491961717605591, 1.2778372764587402, 0.4495754539966583, -0.41418683528900146, -0.5435444116592407, -0.2313576191663742, 0.7080010771751404, 0.6788956522941589, -0.03437753766775131, -0.09653575718402863, -0.44115108251571655, -0.4893222451210022, 1.3208186626434326, 1.7987874746322632, 0.8000690340995789, -0.11700908094644547, -1.481109380722046, -0.015241440385580063, -0.16477182507514954, -2.192368268966675, -0.6219993829727173, 0.16376648843288422, 0.7510743141174316, 0.4181540012359619, -0.11382555216550827, -2.00042986869812, 0.5193500518798828, 1.2576500177383423, 1.6577402353286743, 0.8708385825157166, -0.8686188459396362, 0.5687186121940613, -0.6964444518089294, 1.051090955734253, 0.4212965667247772, -0.7048578262329102, -1.2278562784194946, -1.4474807977676392, 0.378044456243515, 0.4144945442676544, -0.7025991678237915, 0.2514025866985321, -1.503019094467163, -0.5621694922447205, -0.20621372759342194, 0.3302309811115265, -0.11628852039575577, -0.2752911150455475, 0.9942213892936707, 1.75813627243042, -0.3501875400543213, -0.601731538772583, 0.44714120030403137, -0.8306583166122437, -0.5999729037284851, 1.4778097867965698], [-0.16419106721878052, -0.4954470992088318, -0.35349801182746887, -1.5290946960449219, -1.3574124574661255, 0.6586084365844727, -0.7441736459732056, 0.659426748752594, 0.1331118494272232, -0.9160357117652893, 0.9004057049751282, -0.013711711391806602, -0.29928603768348694, 1.4109469652175903, -0.07249501347541809, -0.5189962983131409, -0.16059449315071106, 0.6182286739349365, 1.3111144304275513, 1.0917754173278809, 1.1430917978286743, -0.8236160278320312, 3.054002285003662, 0.30365610122680664, 0.455392986536026, -0.7718530297279358, 2.0308523178100586, 1.8636773824691772, -1.6694433689117432, -0.09046895802021027, -1.6509233713150024, -0.01130887120962143, -0.7416080236434937, 0.6137468218803406, -0.147908553481102, 0.5429269671440125, -1.8985648155212402, -0.013161590322852135, -0.42734041810035706, 0.13221651315689087, 1.1866811513900757, -0.23635628819465637, 0.4918438792228699, 0.29244333505630493, -0.2603217363357544, -0.09766535460948944, 0.2017715722322464, 0.2722053527832031, 1.0300017595291138, 0.20048080384731293, 0.22282128036022186, 0.30946794152259827, 0.5732274055480957, 0.24327950179576874, 0.43603187799453735, -1.7079951763153076, -0.0024466542527079582, 0.5710340738296509, 0.6276984810829163, 0.6625825762748718, -2.3818249702453613, -0.6392351388931274, 0.008421416394412518, 0.3636913001537323, -0.11440177261829376, 0.7655221223831177, -1.376967191696167, 0.7633156776428223, 0.5862299799919128, 0.696473240852356, 2.4639790058135986, 0.2307230532169342, 0.6646438241004944, -0.2694985866546631, 0.20185503363609314, 0.5449462532997131, 1.3492448329925537, -0.549990177154541, -0.5346759557723999, 0.9620906114578247, 0.19361937046051025, 0.15872347354888916, -1.4184588193893433, 1.3129758834838867, -2.9316396713256836, -1.6904451847076416, -0.32519325613975525, 0.2145494818687439, -0.2676584720611572, -0.6235032677650452, -0.7071678042411804, -0.6762033104896545, -0.7709305882453918, 0.5879572629928589, 0.2988879680633545, 3.2708523273468018, 0.3429909348487854, -0.75005704164505, 0.3490079343318939, -0.3351422846317291, -0.5593899488449097, -2.1389122009277344, 0.6163609623908997, 0.40855705738067627, 1.6565815210342407, 0.45864710211753845, -0.7450583577156067, 1.35478937625885, -1.1741386651992798, -0.700772762298584, -0.4114452004432678, -0.11750221997499466, 0.30426084995269775, -1.3923834562301636, -1.4175599813461304, 0.7798365354537964, -0.9542174935340881, 0.3421267867088318, 2.4094462394714355, 0.18552841246128082, -1.444615364074707, 0.7034654021263123, -0.6612462401390076, -0.4392918050289154, -0.06525634974241257, -0.10661446303129196, 0.8905941247940063, 0.38612109422683716], [1.174759864807129, -0.1984720230102539, 0.385995090007782, -0.8302868604660034, -0.6941043138504028, 2.1326818466186523, 1.2812926769256592, -0.15716488659381866, 1.0417661666870117, -0.6827642917633057, -1.184313178062439, 1.3841757774353027, -1.0075595378875732, -1.1377229690551758, 0.48720842599868774, 0.7516700029373169, 1.093976616859436, -1.0742496252059937, 0.6317984461784363, 0.7201977968215942, -1.437820315361023, -0.5447390675544739, 0.08347184956073761, -0.3323792815208435, 1.2349088191986084, 0.2629074454307556, 0.7131466865539551, 1.493390679359436, -0.9306601285934448, 0.4703596532344818, -0.26388585567474365, 1.0463602542877197, -1.3346832990646362, -0.9332764148712158, 0.6409483551979065, 0.34488558769226074, 1.1018198728561401, 1.4693055152893066, 0.3738521635532379, -1.0949499607086182, 0.15706782042980194, -0.12404856830835342, 1.430465579032898, -0.1792353391647339, 1.4474257230758667, 0.3656076490879059, 0.659922182559967, -0.7509600520133972, 1.0631154775619507, 0.15184341371059418, -1.0311006307601929, -0.8767987489700317, 0.2058938443660736, 0.2412225753068924, 3.2590408325195312, -0.8327195048332214, 0.34170782566070557, 1.2927982807159424, 0.30893653631210327, -0.059440359473228455, -0.7294021248817444, 0.647040069103241, 0.47898632287979126, 1.3962088823318481, -0.17044876515865326, 2.3875248432159424, -1.3473341464996338, -0.9940778017044067, -0.3452073931694031, 0.05276072397828102, -0.20934122800827026, 1.6915092468261719, 0.01819867081940174, 0.8297351598739624, -0.556074857711792, 2.126052141189575, -0.48689326643943787, 0.18012377619743347, 0.5916167497634888, 1.1065168380737305, -0.8272897005081177, 0.8507338762283325, -0.5355073809623718, -2.106796979904175, -0.7886239290237427, 0.3016127347946167, -0.05214795842766762, 0.21304093301296234, 0.0900835245847702, -1.583509087562561, -1.2612844705581665, 0.8206032514572144, -1.1053521633148193, -0.9710742235183716, 1.1432067155838013, -0.14502878487110138, -0.2814350426197052, -0.06327197700738907, -0.15616987645626068, 0.4810309112071991, 0.19095838069915771, -1.2500962018966675, 0.38446110486984253, -0.06101750209927559, 0.4161451756954193, -0.7818183898925781, -0.6955012083053589, 0.04417579248547554, -0.02401597611606121, 0.33527228236198425, 1.4900412559509277, 0.4902579188346863, 0.6866147518157959, 0.8599972724914551, -2.021524667739868, 0.0257373359054327, -1.5631173849105835, 1.3566200733184814, -0.4182371497154236, 0.8369268178939819, 0.28893184661865234, 1.0473659038543701, 0.46547064185142517, 0.20814457535743713, -1.1208105087280273, 0.767794668674469, -1.3985697031021118, -0.585385799407959], [0.5119112730026245, 0.1858423799276352, 1.4952304363250732, 0.8845105171203613, 1.196502447128296, -1.1037894487380981, 0.4628582000732422, -0.46977895498275757, -0.7602809071540833, -0.5695205330848694, -1.6978200674057007, -0.020779838785529137, 0.45245352387428284, 1.6213387250900269, -0.38692864775657654, -0.2964158356189728, -0.8187623620033264, 1.3023430109024048, -0.4101093113422394, -1.2204948663711548, -0.7198843955993652, 0.9158459901809692, 0.45398661494255066, -1.3548556566238403, 1.026632308959961, 1.8454399108886719, -0.6410746574401855, 1.0584908723831177, -0.041016556322574615, -0.6031590700149536, 0.10378996282815933, -0.9527600407600403, 0.20981557667255402, -0.23626989126205444, -0.3291676640510559, -0.2007828801870346, -0.6380840539932251, -1.560124158859253, 1.8832541704177856, 0.45240339636802673, 0.16392286121845245, 0.38462895154953003, -0.2700069844722748, -0.021158168092370033, -0.4203889071941376, 1.7086312770843506, -0.35839158296585083, -0.6918678283691406, 2.28501296043396, -2.292022943496704, 2.2002370357513428, -0.4719434380531311, -1.2056820392608643, -1.525365948677063, -0.43403157591819763, 1.076427698135376, -0.849483847618103, -0.12276807427406311, 0.12943711876869202, -0.5848707556724548, -0.46553558111190796, -0.44665831327438354, -0.7270340919494629, -1.416480302810669, 2.160862684249878, -0.36056655645370483, 0.1605435013771057, -0.8887892365455627, -0.5086060166358948, -1.4552644491195679, -1.905051827430725, -1.4760712385177612, -0.8503946661949158, 1.1321501731872559, 0.919573187828064, 0.5511624217033386, -0.8751561045646667, -0.22043006122112274, 1.9542313814163208, -1.0532432794570923, 1.037422776222229, -1.4614195823669434, -0.7049126029014587, -2.6486024856567383, -0.6281515955924988, -0.0023272691760212183, 1.3815349340438843, -3.0308611392974854, -1.3956671953201294, 0.7377316951751709, 0.05867486447095871, 0.1571967601776123, -0.6744539141654968, -0.5090218782424927, -1.1022318601608276, -0.9083011746406555, 0.43486514687538147, 0.715440571308136, -1.0038959980010986, -0.28802183270454407, 1.5611048936843872, 2.330977201461792, -2.5836479663848877, 0.4398203492164612, 0.569857120513916, 2.1778016090393066, 0.7585882544517517, 0.8557271361351013, 0.24795827269554138, -0.5007612705230713, -0.6708087921142578, 0.49653029441833496, 1.1560847759246826, -0.5705828666687012, 0.9150184392929077, 0.46814823150634766, -0.6923663020133972, 0.7890945672988892, -1.305985450744629, 0.7394944429397583, 0.35318711400032043, 0.27722984552383423, -2.894261360168457, 0.8975138068199158, -0.6502333283424377, -0.11400812864303589, -0.3109816014766693, 0.2822537124156952], [0.46927163004875183, 1.8690553903579712, 1.0492113828659058, 0.744422435760498, 1.3767508268356323, -0.18909721076488495, -1.074660301208496, 0.9683603048324585, 0.7182703614234924, 0.9544312357902527, -0.23459777235984802, -0.6770216822624207, -1.0201481580734253, -0.6292394399642944, 0.22813721001148224, 2.0185225009918213, -2.2383837699890137, -1.8975393772125244, -0.6292702555656433, -0.7500602602958679, -1.1120374202728271, -0.8402475118637085, 0.3302463889122009, -0.05853066220879555, 0.14850692451000214, 0.5995717644691467, -0.6567916870117188, -0.9265908002853394, 0.02871061861515045, 0.11069999635219574, 1.9694323539733887, -0.06691192090511322, -1.0263334512710571, 1.2092430591583252, 0.7832869291305542, 1.8593133687973022, 0.26408106088638306, -1.0238409042358398, -1.5490880012512207, 0.6434218883514404, 0.33023980259895325, 1.0859028100967407, -0.2687842547893524, -0.7585583925247192, 0.454622358083725, 0.20932969450950623, 1.5164326429367065, 0.8029947876930237, -0.01618974842131138, 0.21770870685577393, -0.6704525351524353, 1.1120949983596802, -0.29464879631996155, 0.6853474378585815, 1.6723275184631348, -0.24909129738807678, 0.33277276158332825, -0.9442580938339233, 0.6683734059333801, -0.4113752245903015, 0.6452982425689697, 0.3890533745288849, 0.09034127742052078, -0.287288635969162, 0.22820530831813812, 0.5231653451919556, 1.7756433486938477, -1.1553806066513062, -0.20684143900871277, -1.141376256942749, 1.37961745262146, -0.4248732328414917, -0.46540188789367676, -0.21431000530719757, 1.9746648073196411, 1.5043045282363892, 1.2434146404266357, 0.9513189196586609, -0.2904481291770935, 1.0088093280792236, 1.4800347089767456, -0.9422439932823181, -1.4502606391906738, 0.7248851656913757, 0.17909571528434753, 1.5965297222137451, -0.1169627457857132, -0.012959972023963928, 1.0851974487304688, -0.7583026885986328, -1.3521969318389893, -0.7823224067687988, -0.3470476269721985, 0.7221280336380005, -0.33223432302474976, -1.4133274555206299, -0.03269541263580322, 1.4057071208953857, 0.9005226492881775, 0.8174678087234497, 0.056115634739398956, 0.0916285514831543, 0.02276415005326271, -0.6833791136741638, 0.7832762598991394, 0.8975976705551147, 0.1865101158618927, 0.3734245002269745, 1.695228099822998, 1.642382264137268, 1.265574336051941, 2.930586814880371, 0.5910797715187073, 1.8891762495040894, 1.1661815643310547, -0.8341478705406189, -0.740534782409668, -0.058793313801288605, 0.5066565871238708, -0.6464918851852417, 0.5057239532470703, -0.2696519196033478, -0.613659143447876, -0.6871113777160645, -0.42102205753326416, -1.1390208005905151, -0.0529961884021759, 0.36811739206314087], [-0.14702029526233673, 1.0128514766693115, -0.3924228250980377, 0.4613098204135895, 0.6891602277755737, -0.5185005068778992, 0.8655959963798523, -0.11850766092538834, -1.2878456115722656, 0.4032723605632782, -0.906770646572113, -0.6544309854507446, 0.03137670457363129, -0.6875666379928589, 0.9308204650878906, 0.7035338878631592, 0.2706095576286316, 0.2535277009010315, 1.554535984992981, -1.9194530248641968, 0.04473498836159706, -1.1205836534500122, -2.4980549812316895, -0.27179673314094543, 0.019649427384138107, 0.9155442118644714, -1.254833459854126, 0.06621558219194412, -1.350703477859497, 0.29994630813598633, -0.14659711718559265, 0.895682156085968, -0.008548124693334103, 0.12570449709892273, 0.7922096848487854, 1.4986213445663452, 0.38307783007621765, 0.8606810569763184, -1.1366349458694458, -0.6281520128250122, 1.1469711065292358, -0.207772359251976, -0.8754019737243652, -2.412644386291504, -0.30143240094184875, -0.25813907384872437, 0.403361439704895, 0.4182296395301819, 2.375504493713379, -0.9174664616584778, 0.7346527576446533, 0.3273985683917999, 1.9422045946121216, 0.26007911562919617, -0.179824560880661, -1.391295075416565, -0.14644113183021545, 0.9431186318397522, 0.875810980796814, 1.4857313632965088, 0.10013856738805771, -0.11197565495967865, 0.9350696802139282, 0.3700796365737915, 0.19678093492984772, -0.19682283699512482, 1.6803159713745117, 0.6591100692749023, -1.0332387685775757, -1.7855546474456787, 0.23285096883773804, -0.6011430025100708, -1.5139843225479126, -0.08298465609550476, 0.7527727484703064, -0.9191069006919861, -1.911931037902832, -0.4735759198665619, -0.8948176503181458, 0.7968233823776245, 0.8248979449272156, 0.37423616647720337, -1.1489850282669067, 1.268758773803711, -2.297626256942749, -0.6470083594322205, 0.2603084146976471, -0.587441623210907, -0.4976119101047516, 0.6746240854263306, 0.8751455545425415, 0.42766109108924866, -0.24991093575954437, -0.44558921456336975, -0.3390774130821228, 0.7898320555686951, 2.454477310180664, 2.1697428226470947, -0.6255019307136536, -0.8752427697181702, 0.7274616360664368, 0.6615381240844727, -0.8760125637054443, -0.6250908970832825, 0.4490675926208496, -1.6489886045455933, 0.4417370855808258, -1.2341389656066895, 1.2524793148040771, 1.5914225578308105, -0.743373453617096, 0.37878769636154175, -1.6371065378189087, -0.6369242668151855, 0.31656166911125183, 0.6350380778312683, 1.539554476737976, 1.8021502494812012, 0.5367295145988464, -0.49900686740875244, 2.2458178997039795, -1.6681995391845703, 1.9227842092514038, -0.5189853310585022, -1.9371732473373413, -0.8844345211982727, 0.3631167411804199, -1.3518691062927246], [-0.3797996938228607, 0.04672596603631973, 0.1654200404882431, 0.12404938787221909, 1.1120432615280151, -0.34677597880363464, 0.792881190776825, -0.2706851661205292, 0.16098900139331818, 1.963648796081543, -1.216409683227539, -1.1364307403564453, -0.8090731501579285, 0.4010382890701294, -0.8093071579933167, 0.09376917034387589, 1.0615572929382324, 0.8486170172691345, 0.30404335260391235, -0.3552376925945282, -0.385096937417984, 0.7852075695991516, 0.8895736932754517, 1.9508470296859741, -1.1302837133407593, -0.0060152155347168446, -1.5811213254928589, -0.8170989155769348, -0.6838355660438538, 2.1866190433502197, 0.032309141010046005, 0.19338101148605347, -0.8849688172340393, 0.40393292903900146, 1.461477279663086, 0.1522563397884369, 0.741371214389801, 0.6108025908470154, -0.8873277306556702, -1.795432209968567, -0.5824406743049622, -1.6280947923660278, -0.09116023778915405, -0.4648681879043579, -0.5117747783660889, -0.0008459393284283578, -0.6890935301780701, -1.3768476247787476, 0.482242226600647, -0.9615782499313354, -0.16889208555221558, -0.48023656010627747, 0.3437390625476837, -0.10016267746686935, 0.13850027322769165, -0.8256372809410095, -0.9903673529624939, -0.6693705916404724, 0.13874255120754242, 2.634347677230835, -0.5287542343139648, 3.3245503902435303, 0.09814896434545517, 1.300255298614502, -0.5089761018753052, 1.2599399089813232, -1.6572999954223633, 0.400902658700943, -0.012968763709068298, -0.25930115580558777, 0.5444729328155518, 1.5245941877365112, -1.8745105266571045, 1.237709641456604, -1.1347346305847168, 0.6340543627738953, -0.09563414007425308, -1.1280267238616943, 0.6901901364326477, 1.0906919240951538, 1.0571997165679932, 0.9740741848945618, -0.45283544063568115, -0.14687781035900116, 0.34649428725242615, -0.11812590062618256, 1.4145622253417969, 0.7391951084136963, -0.11985091120004654, -1.274863600730896, 1.3426200151443481, -0.44186922907829285, 0.9208256602287292, 0.8876538276672363, -0.17346252501010895, -0.5980551838874817, -1.8602668046951294, 0.12048730254173279, -0.08416365832090378, -0.6785548329353333, 0.4629162847995758, -1.3434659242630005, 0.2629845440387726, 0.034589242190122604, 0.41716375946998596, 0.7096508741378784, -0.6343430280685425, -0.3829081654548645, -2.98742938041687, -0.954781174659729, 0.9556785225868225, -2.5250320434570312, 0.4002355933189392, 0.5876076221466064, -2.4916772842407227, -1.5223060846328735, -1.023024320602417, -1.1903291940689087, 2.131653070449829, 1.2502332925796509, 0.4304506778717041, -1.5533571243286133, 0.9748517870903015, 1.6880964040756226, 0.19281059503555298, 0.5787075757980347, -1.644182801246643, 1.3628023862838745], [-0.4950946271419525, -0.0004177975933998823, 0.4853728711605072, 0.2325759083032608, -0.2690838873386383, -0.0495508573949337, -0.19517715275287628, -1.7813740968704224, 0.4237848222255707, -0.1887662559747696, -0.9167870879173279, 1.6638946533203125, 0.3169313967227936, -1.1228559017181396, -0.839691698551178, -1.2867571115493774, -0.5379772782325745, -1.9740421772003174, 0.0016549912979826331, 0.7742971181869507, 0.1640566885471344, -1.890791654586792, -0.8907774686813354, 0.2980172336101532, -1.5142359733581543, 0.9156391620635986, 0.034862976521253586, -0.08637484908103943, 0.1293305903673172, -0.06761161237955093, 0.3141286075115204, 0.018372556194663048, -0.08011100441217422, -1.8054825067520142, -0.33279454708099365, 0.5908327698707581, 1.3205313682556152, 0.5483512282371521, -1.0962090492248535, 1.353469967842102, 0.7803504467010498, 0.15214377641677856, 1.066481113433838, -0.33543333411216736, 0.6539960503578186, 0.24602828919887543, -0.35187336802482605, -0.6732770800590515, -0.4641341269016266, 0.07182036340236664, -0.19226722419261932, -1.3575894832611084, -0.5426420569419861, 1.4731467962265015, -0.2408226728439331, 0.48216772079467773, 0.4484650194644928, -0.4889819324016571, 1.1846669912338257, -0.6851280927658081, 0.6051470041275024, 0.28313252329826355, -1.559629201889038, 0.6174116730690002, 0.18413525819778442, 0.7427496314048767, 0.17844781279563904, 0.522068440914154, -0.15413568913936615, -1.9854702949523926, -0.2777999937534332, 0.3117562532424927, 1.0787875652313232, -0.6498361825942993, 0.07345988601446152, 1.5145901441574097, 0.19838304817676544, 0.4048132300376892, -0.4355015456676483, -2.5730819702148438, 0.6360929012298584, -0.6791548132896423, 1.6898434162139893, -1.082094430923462, -0.552993893623352, -0.48438042402267456, 1.157284140586853, -0.5402846932411194, 1.1045777797698975, -1.5429500341415405, -0.3245462477207184, -0.1371874213218689, 0.20210671424865723, -0.5299212336540222, 1.1708579063415527, 0.742716372013092, -0.2128778100013733, 0.7323403358459473, 0.2989102900028229, 0.41249513626098633, -0.9746181964874268, -0.053099341690540314, 0.9857181310653687, 0.6507065892219543, -0.7957800626754761, -0.4367348849773407, 0.20319864153862, -0.8838422894477844, 0.8275958895683289, 0.7688368558883667, 0.0874231681227684, 1.6025060415267944, 1.9582104682922363, 1.4738200902938843, 1.6673641204833984, -1.9433777332305908, -0.23065149784088135, 3.0423076152801514, 1.1882632970809937, -1.3925631046295166, -0.030787711963057518, 0.43055713176727295, -0.07672511041164398, 0.4814363718032837, 1.2635838985443115, 1.0232008695602417, -0.8297560811042786, -1.0390795469284058], [-0.9218165874481201, 0.1677422970533371, 0.44541311264038086, 0.5686929821968079, 0.8931602835655212, -1.9645205736160278, -0.9258341193199158, 0.9205254316329956, -0.21660946309566498, 0.9249923229217529, -0.9719873666763306, -0.2672490179538727, 0.13749843835830688, 1.7339510917663574, 2.027453899383545, -0.7339910268783569, -1.176265001296997, 1.2491810321807861, -2.0773603916168213, -0.4852023422718048, -0.9214985966682434, -0.13069526851177216, -2.6302592754364014, 0.47087952494621277, 0.2784951627254486, 0.7140417098999023, 1.7132927179336548, 0.3245837688446045, -0.22577925026416779, 0.1785568743944168, 1.0433244705200195, -1.6996958255767822, 0.9108125567436218, 0.8036385178565979, -0.5424439907073975, -1.2183090448379517, -0.312560111284256, -0.31301453709602356, 0.5440047979354858, 0.05789845064282417, 0.8603134155273438, 0.8971387147903442, -0.31357958912849426, 0.37471917271614075, -0.20984388887882233, 0.2886489927768707, -1.011025071144104, -0.024114646017551422, 0.5570234656333923, -0.06277686357498169, 0.7401538491249084, -0.735760509967804, 1.5238951444625854, 0.2420387715101242, -1.8158583641052246, 2.027813673019409, 0.1343178004026413, 1.227086067199707, 0.11069610714912415, 0.0834704339504242, -1.0037111043930054, -1.3031644821166992, 0.15177255868911743, -0.19295229017734528, -0.7462527751922607, 1.5496773719787598, -2.118103504180908, 0.2438737452030182, 1.069764494895935, -1.746843695640564, 0.38340651988983154, 1.3601770401000977, -0.4571692645549774, 0.10407284647226334, 2.059312343597412, 0.8341892957687378, 0.4722750186920166, -0.24100269377231598, 0.42222973704338074, 0.24665147066116333, -0.052300143986940384, 1.6491676568984985, 1.5438899993896484, 1.3746058940887451, -0.14744488894939423, 0.39159098267555237, -0.4598709046840668, 0.08316569030284882, 0.40615352988243103, 0.34323468804359436, 0.19729307293891907, -2.799193859100342, -2.0246071815490723, -0.16470898687839508, -0.12948763370513916, 1.251677393913269, 0.2039993852376938, -1.1064025163650513, -0.279745489358902, 2.277400493621826, 0.12749779224395752, -0.10745526105165482, -0.9265866279602051, 0.8924151659011841, 1.0464119911193848, -1.0488829612731934, -0.06552179157733917, -0.3965306580066681, -0.057312387973070145, 0.23733606934547424, 0.46381646394729614, -0.057660095393657684, -1.5479165315628052, -0.004524436313658953, -1.5195834636688232, 1.0632694959640503, -0.04095444083213806, -1.0716543197631836, -2.258967876434326, 2.0843870639801025, 0.9411816000938416, 0.20698858797550201, 0.7448492050170898, 0.5913076996803284, -0.49657973647117615, 1.621703863143921, -0.4577363133430481, -1.56039559841156], [0.2616744935512543, 1.736837387084961, -1.0656123161315918, 0.6116881966590881, 2.410872459411621, 0.4058357775211334, -0.20241469144821167, 0.9793832898139954, -0.4617888927459717, -0.22276301681995392, -1.860129475593567, 1.1641751527786255, 0.4829307496547699, -0.9816942811012268, -0.011044648475944996, 0.3145337998867035, 1.187056541442871, -0.18357062339782715, -1.4430820941925049, 0.6708617806434631, -1.9650380611419678, 0.14819936454296112, -0.09082812815904617, 0.22832277417182922, -0.4853837490081787, -0.4996732175350189, -0.16749154031276703, -0.6066550016403198, 0.14202745258808136, -0.3856724500656128, 0.38388270139694214, -1.3395568132400513, 0.9981274604797363, 0.7048019170761108, 0.01440469454973936, 0.1726580411195755, -0.08378534018993378, 1.220146656036377, -1.306082010269165, 2.0344340801239014, 0.12448059767484665, -0.8956615924835205, 1.0317726135253906, -0.26740536093711853, 0.6027617454528809, 0.1228393092751503, -0.42287516593933105, 0.5601435303688049, 0.558279275894165, -1.6026852130889893, 1.340398907661438, 0.2153925895690918, 0.5063785910606384, 0.5325232148170471, -0.30542656779289246, -2.7751200199127197, 1.1579855680465698, 2.1189205646514893, 0.5333417654037476, -0.8128756284713745, 0.7983343601226807, 0.22244340181350708, -1.3121709823608398, 0.4063725173473358, -0.35183337330818176, 2.785097122192383, 1.1857324838638306, -2.3581881523132324, -1.849977970123291, 1.0235592126846313, 0.6791069507598877, 0.1629190444946289, 0.10525845736265182, 1.781282663345337, -1.1300911903381348, 1.186409831047058, -0.2583260238170624, 1.4417188167572021, -0.016600288450717926, 0.40246716141700745, 1.315965175628662, -0.6944981813430786, 0.9441971778869629, -0.37749582529067993, -0.9634283781051636, 0.95501309633255, 0.5468873977661133, -0.9429483413696289, 0.574668824672699, -0.32064464688301086, 0.21817271411418915, 0.8192757368087769, -1.1361744403839111, 0.19760040938854218, 0.8510323762893677, 0.023549268022179604, 1.1964952945709229, 0.7310499548912048, -1.7291051149368286, 0.23577891290187836, -0.7114573121070862, 1.0266156196594238, 1.1250168085098267, 1.3180838823318481, 0.8656696081161499, -0.07826174795627594, 0.06625103950500488, -0.3191241919994354, 0.6328321695327759, 0.3419477641582489, -0.2996852695941925, -0.7273017168045044, 0.1751072108745575, 1.5399707555770874, -2.3339500427246094, -1.657256841659546, -0.0008211311651393771, 1.7137683629989624, 0.07396300882101059, 0.06518423557281494, 1.2403806447982788, -1.1573271751403809, -0.23152102530002594, 0.02853071503341198, -0.7884434461593628, 0.4929777979850769, 0.2767540514469147, -1.0196213722229004], [-0.34908801317214966, -1.2609148025512695, 0.5712495446205139, -0.9024013876914978, 2.2623002529144287, 1.3833770751953125, 1.3148287534713745, 0.2957204282283783, 0.36106693744659424, 2.388068675994873, -0.2087382972240448, -0.3142123818397522, -0.3415184020996094, 1.052299976348877, -0.44132542610168457, -0.8630778789520264, -1.723085641860962, 1.712479829788208, 0.2047816663980484, -0.9508550763130188, 0.05687698349356651, -1.584736704826355, -0.35181286931037903, 0.3058631718158722, 0.6348018050193787, 0.7653147578239441, 2.1686646938323975, -0.26305684447288513, 0.69559645652771, 1.232000708580017, -1.4884889125823975, -0.8050734400749207, -0.4514578878879547, 0.3415572941303253, 0.2361767590045929, 0.12103884667158127, 0.9384494423866272, -0.19991356134414673, 2.030000925064087, 0.008660249412059784, 0.5379356741905212, 0.8605815768241882, 0.1514187902212143, 0.2685176730155945, -1.2772003412246704, 0.6341232657432556, 2.1570000648498535, -0.780875027179718, 0.40749120712280273, -0.885951042175293, 0.05587344989180565, 0.5991289019584656, 0.5096819400787354, 0.753214955329895, 0.74298495054245, -0.5586791038513184, 0.1528160125017166, -0.37593674659729004, 0.7055867910385132, 0.20246991515159607, 1.4516772031784058, -1.7483797073364258, -0.217364102602005, -0.012688534334301949, -0.3679661452770233, 1.269838809967041, 0.8200747966766357, -0.46216854453086853, 0.03192444145679474, -0.05430218204855919, -0.4578405022621155, 0.37059128284454346, -0.7916317582130432, 1.3918561935424805, -0.7114278674125671, -0.4097236692905426, 1.6067661046981812, -0.7332808971405029, 0.23696550726890564, 0.6391826272010803, 1.8561248779296875, 0.6168848872184753, -0.06872525811195374, 0.4887254536151886, -1.2327853441238403, -0.0706239640712738, 0.6727572679519653, -0.39427095651626587, -1.544653296470642, 0.3329317569732666, 0.40587934851646423, 1.564347267150879, 0.4223422706127167, 0.4319460093975067, -0.5367231965065002, 0.4348623752593994, 1.01616632938385, -0.30951035022735596, 0.7940919399261475, 1.1767176389694214, 0.2829647362232208, 0.7711350917816162, -0.2642783224582672, -0.9301932454109192, 0.6368077397346497, -0.5615721940994263, 1.0891039371490479, -1.428057312965393, -1.3791089057922363, 0.8687558174133301, 0.5746719837188721, 1.0360586643218994, -1.8390852212905884, 0.5697829127311707, -2.238032579421997, 0.8328979015350342, -0.5083702802658081, -0.7059940099716187, 0.8638206720352173, -1.718570351600647, 1.3377782106399536, 0.878829836845398, 0.47993606328964233, 0.7924442887306213, -0.34504371881484985, 1.1951907873153687, 1.334449291229248, 0.02670002356171608], [-0.9811322689056396, -0.5764033794403076, -0.6528123617172241, 0.354862242937088, -1.7469004392623901, 1.2495648860931396, 0.44025978446006775, -0.39504802227020264, 0.31542670726776123, 0.08028919249773026, -0.39731475710868835, -0.21533365547657013, -0.19544506072998047, -0.11071603745222092, -0.4464816153049469, 0.024690546095371246, 1.2457170486450195, 0.3667176067829132, 0.4641343057155609, -0.19781456887722015, -0.14683039486408234, 0.3172343671321869, -0.8712732195854187, 0.45373353362083435, 0.1150873601436615, 0.40425559878349304, -0.8693758249282837, 0.6580507159233093, -0.4145728051662445, -1.3120031356811523, 1.4684700965881348, -0.29617905616760254, -0.8429701328277588, -0.5484756231307983, -1.0131299495697021, 0.8655506372451782, -0.3254447281360626, 1.1215559244155884, -1.165248155593872, -0.3378376364707947, -0.2715816795825958, -1.294811725616455, 0.8735565543174744, -0.4550440013408661, 1.1966524124145508, -0.5059733986854553, 1.1936780214309692, 0.15142981708049774, 0.1957734227180481, 0.38184601068496704, 0.5813747048377991, 1.2673072814941406, 1.9630558490753174, -0.5035953521728516, -0.025816895067691803, -0.08696535974740982, 2.054208517074585, 0.46386605501174927, 1.024585247039795, 0.32554173469543457, -0.547754168510437, -1.0734825134277344, -0.39154288172721863, -0.8346596956253052, -0.8423298001289368, 0.10170026123523712, 0.07751930505037308, 2.3810160160064697, -0.44728097319602966, 0.31983160972595215, 1.1002670526504517, 1.8098431825637817, 0.4701406955718994, 0.23249772191047668, -1.242488980293274, 0.5980486869812012, 1.0137492418289185, -1.3147319555282593, -1.2479561567306519, 0.8560728430747986, 0.4353930354118347, 0.667086660861969, 0.657637357711792, 2.69435977935791, 1.6849391460418701, 1.4491328001022339, 0.9153436422348022, 0.8264410495758057, -0.34166407585144043, -0.269552081823349, -0.8879337310791016, 0.4639374911785126, 0.4492855966091156, -1.1195237636566162, -1.4014490842819214, -0.2117409110069275, 0.0476699098944664, 0.8756779432296753, 1.1147313117980957, 0.2531118392944336, -0.749271810054779, 0.4111400842666626, -0.07700031995773315, -0.13011176884174347, -3.418560028076172, 0.27005109190940857, -1.3495609760284424, 0.9425268769264221, -1.473161220550537, 0.5318987965583801, 0.11142431199550629, 0.03301934152841568, -0.026848524808883667, -0.8701158165931702, -1.454253911972046, 1.6608444452285767, 0.40154021978378296, -0.939113974571228, 3.6132559776306152, 1.0713739395141602, 1.0834846496582031, -0.39820584654808044, -0.7006902098655701, -1.6862142086029053, -0.2032414972782135, 1.5657843351364136, 1.1132657527923584, -0.5988068580627441]], [[-0.49508288502693176, 1.1328749656677246, 1.3961580991744995, 1.230374813079834, -0.9251841306686401, 0.29293227195739746, -0.0979001373052597, -0.41319185495376587, -2.6353585720062256, 0.4835047125816345, -1.1814334392547607, -0.04407177120447159, -0.33959072828292847, -0.4905139207839966, 0.3091663718223572, 0.8126451373100281, -0.8655591011047363, -1.962278962135315, -1.5411205291748047, 0.12598948180675507, 0.8272179961204529, -1.5545212030410767, -0.9311359524726868, 0.7698749303817749, 0.7102542519569397, -0.7721611261367798, -0.29180586338043213, -1.2942456007003784, 1.126670002937317, -0.007218505721539259, 0.7374448180198669, -0.006188973784446716, -0.8474590182304382, -1.4374948740005493, 0.3043502867221832, -0.39829808473587036, 1.3979525566101074, 1.0890761613845825, -0.4569302499294281, -0.7945294380187988, -0.7869699001312256, -0.37200191617012024, 0.891444206237793, 2.0983407497406006, 2.020233392715454, 0.23546788096427917, 1.457056999206543, 0.1935955137014389, -0.11112222075462341, -0.8258165121078491, -1.0002063512802124, 2.88533091545105, 0.807851254940033, 0.7057927250862122, -0.639349102973938, -0.17732112109661102, 0.29079535603523254, 0.4441201090812683, -0.5541505813598633, -0.8992765545845032, 0.1030532717704773, -0.20933641493320465, 0.7435040473937988, -0.8459417223930359, -0.35049381852149963, -0.722560703754425, -1.9794631004333496, 0.6809599995613098, 0.8093897104263306, 0.1217445433139801, 0.79069584608078, -1.0355039834976196, -1.1283502578735352, -1.5865237712860107, 0.7481391429901123, 1.5504463911056519, 0.5887798070907593, -0.6836836934089661, 0.8930568695068359, 2.286825180053711, -0.3293077349662781, -0.1745598167181015, 0.4756610691547394, 0.15822264552116394, 0.08564150333404541, -1.7511003017425537, -1.0333608388900757, -0.9374071359634399, 0.6729174256324768, -0.4945610761642456, 1.32024347782135, 1.7613744735717773, -0.8616304993629456, -3.01060152053833, 0.6972779631614685, 0.8262752890586853, 0.7143570184707642, 0.75553959608078, 0.8478005528450012, -0.904625654220581, 0.3009980320930481, 1.445870280265808, 0.9707282185554504, -0.6982276439666748, -0.6851001381874084, -0.6838979721069336, -0.11954076588153839, -0.21257276833057404, 1.5642647743225098, -0.8091035485267639, -0.17242421209812164, 1.0082457065582275, 1.04649817943573, 0.9664633870124817, 0.16304869949817657, 0.6702221035957336, -1.1253762245178223, -0.0398918017745018, -1.1111531257629395, 1.4034626483917236, 1.0416926145553589, 1.814111590385437, 0.15791639685630798, 1.41635000705719, 0.6163884997367859, -0.5827382206916809, 0.673616886138916, 0.6523537039756775], [1.7571483850479126, -0.144189715385437, -0.6776203513145447, -0.17584601044654846, 0.2795966565608978, -0.9190607666969299, 1.0173300504684448, -3.4688568115234375, 1.6697285175323486, 2.4152462482452393, 1.2233240604400635, -0.9601015448570251, 0.5593283176422119, -0.9268233776092529, -2.236603260040283, 0.2217058539390564, -0.8439337015151978, -0.9532984495162964, -0.4202956557273865, -2.337122917175293, 0.31862854957580566, -1.1320080757141113, -1.3186218738555908, -0.03268054500222206, 0.8268275260925293, -2.121828079223633, -0.0612921267747879, -0.2946539521217346, 0.6940603256225586, -0.24481092393398285, 1.098279356956482, -0.3171485662460327, 1.3914697170257568, -0.8934106826782227, -0.619810938835144, 1.2626947164535522, -1.4056575298309326, -0.9962124228477478, -1.4350881576538086, -0.07523766905069351, 0.580938994884491, -0.3681109547615051, 0.30440929532051086, -0.17264492809772491, 0.19737370312213898, 2.098860025405884, 1.8800755739212036, 2.0106863975524902, 0.3954111933708191, -0.26052844524383545, -0.7050338387489319, 1.1604186296463013, 1.1832994222640991, -0.41949227452278137, -0.07686903327703476, 0.809237003326416, 1.648395299911499, -0.1955995261669159, -0.5317882895469666, -0.5641043782234192, -1.2557517290115356, -0.3707937002182007, -0.11095424741506577, -0.5631075501441956, 1.258978247642517, -2.010483741760254, -0.23168422281742096, -0.8271847367286682, 1.0196623802185059, 0.6966983675956726, 0.7153303623199463, 0.35724127292633057, -0.4754006266593933, 1.043428897857666, 0.1699417531490326, -0.032644230872392654, 0.4660240113735199, 0.6243302226066589, 1.2058039903640747, 1.4203200340270996, -1.6840680837631226, 1.3189290761947632, -0.6532136797904968, -0.36365845799446106, -1.6930549144744873, 0.6474118232727051, 0.8760015964508057, 0.8550711870193481, -1.265960454940796, 0.8244131803512573, -0.30187463760375977, -0.0006799378315918148, -1.1550017595291138, -0.7449527382850647, 0.6665607690811157, 0.03477977588772774, -0.007314211223274469, -1.4641213417053223, 0.4210645854473114, 0.00919248815625906, 0.19175465404987335, 0.6483851671218872, 0.27472347021102905, -0.09995579719543457, -1.185138463973999, -0.5139800310134888, 1.5182862281799316, -0.3929627239704132, -0.12400279939174652, 2.7615506649017334, 0.4804028272628784, 0.4166841506958008, 1.213782548904419, 0.7854275703430176, 1.3216909170150757, -0.1148398220539093, -0.5806849598884583, -1.0109328031539917, -1.0693364143371582, 0.1418655514717102, 0.3736174702644348, -1.1492340564727783, 2.394333839416504, -0.6646966934204102, -0.741991400718689, 1.2633662223815918, -0.7153457403182983, -0.471780925989151], [0.9511034488677979, -1.4569872617721558, 1.3490591049194336, -0.6238250732421875, 0.7562414407730103, -1.3268437385559082, -0.945874035358429, 2.2153117656707764, -0.7967441082000732, 0.7484697699546814, -0.4545625150203705, 0.3393722176551819, 1.1188063621520996, 0.10277336090803146, -2.6023876667022705, 1.158372402191162, -0.09991560876369476, 1.4398880004882812, -0.1402847170829773, -0.5015721917152405, 1.1197165250778198, 0.06306006014347076, -1.1568506956100464, -0.36892440915107727, 1.0049724578857422, -0.5139506459236145, -0.022154366597533226, -0.4838860034942627, -0.2635878622531891, -0.7709855437278748, 1.3746429681777954, -1.2088758945465088, -0.5633798837661743, 0.340947687625885, 1.3637686967849731, -2.301226854324341, 0.9077404737472534, 0.8576945066452026, -0.7068519592285156, 1.6514500379562378, -1.165951132774353, 0.625430166721344, -0.21929916739463806, -1.0259926319122314, -0.5824302434921265, 0.37984293699264526, 0.07216043025255203, -1.2306245565414429, 2.3227944374084473, -1.767578125, 0.40340274572372437, 0.9785141348838806, 0.33929574489593506, -1.222662091255188, 0.875041663646698, 0.03324573114514351, -1.1791807413101196, -0.7967506647109985, -0.24432401359081268, 1.2266592979431152, -0.8011435866355896, 0.5348514914512634, -0.906343400478363, 0.44585609436035156, 0.35664939880371094, -0.7670162916183472, -0.21392717957496643, 0.958468496799469, -0.8091232180595398, -0.23188787698745728, -1.1761467456817627, 1.7404799461364746, -0.7738348841667175, -0.1870407611131668, -0.6758869290351868, 0.014321337454020977, -0.9687014818191528, 0.6076318025588989, -0.20655867457389832, 1.2715524435043335, 1.9716107845306396, -0.6231895089149475, 0.6589463353157043, -0.16186675429344177, 0.5452919006347656, -0.23628570139408112, 0.33755823969841003, 0.9945696592330933, -0.22524364292621613, 0.3943175673484802, -1.1644705533981323, 1.166796088218689, -1.699620008468628, -0.046821124851703644, -2.5617518424987793, 0.5739532113075256, -0.8560587763786316, -1.2689769268035889, -1.3386651277542114, -0.3294283449649811, 0.17042163014411926, 0.3137146532535553, 0.08467711508274078, -1.1299399137496948, -1.2217106819152832, -0.8239919543266296, 0.35817432403564453, 1.577905297279358, -2.0781118869781494, 0.9623964428901672, 0.16132965683937073, -1.094744086265564, -1.4583083391189575, -0.6195722222328186, 0.21586638689041138, -1.0020179748535156, -0.1814493089914322, 0.3776662349700928, 0.808404266834259, 0.06436147540807724, 0.042994655668735504, -0.3053033649921417, 2.036292791366577, 0.8024787902832031, -1.3657026290893555, -1.1177995204925537, -0.9476826190948486, -0.343612939119339], [-0.9080019593238831, -0.7085014581680298, -0.15942667424678802, -0.44625696539878845, -1.4412832260131836, 1.9980669021606445, 0.695580780506134, 0.6155415177345276, 1.218881368637085, -1.4199354648590088, 1.5778313875198364, 0.054449040442705154, 0.8398698568344116, -0.48610764741897583, -1.2745096683502197, -0.08457837253808975, 0.0033056605607271194, 0.7542585730552673, -0.8710440397262573, -1.0363099575042725, -0.9151302576065063, 0.10975909233093262, -0.1832091361284256, -1.6664921045303345, 0.23530037701129913, -2.0687286853790283, 0.4722689092159271, 0.3910665512084961, -1.2517441511154175, -1.499729871749878, -0.6617249846458435, 1.115363359451294, -0.2432222217321396, 1.6815886497497559, -0.5591974258422852, -0.3667178750038147, -0.6577770709991455, 0.10294327884912491, 0.3034133017063141, -0.08117537200450897, 0.34931281208992004, 0.3884942829608917, -0.3936682641506195, -0.17506031692028046, 0.9595673084259033, 1.412243366241455, 1.756700873374939, 1.5303925275802612, 0.9736753702163696, 0.8201487064361572, -1.1878389120101929, 1.278456449508667, -1.2469228506088257, -0.27679717540740967, -0.4788593351840973, 1.7245601415634155, 0.6796795725822449, -1.2356212139129639, 0.28026282787323, 0.8354426026344299, 0.7629198431968689, -0.6150855422019958, -1.1926828622817993, -0.755844235420227, 1.3208881616592407, 0.7383714914321899, -0.14667369425296783, -1.8577266931533813, 0.7304806709289551, -0.9790018200874329, -1.617798089981079, 0.1672177016735077, 1.0009108781814575, 0.5794539451599121, -1.3770487308502197, -1.1279523372650146, -1.3636163473129272, 0.5770522952079773, -0.9083077907562256, -0.9344663023948669, -0.25176289677619934, -1.9177168607711792, 0.9162348508834839, 0.23533038794994354, 0.895155131816864, 0.7564346194267273, 0.16169056296348572, 0.8617466688156128, -1.5111836194992065, 1.664398193359375, -1.153154730796814, 1.2884845733642578, -1.5107667446136475, 0.18582570552825928, -1.2223246097564697, 0.3587959110736847, 1.1661783456802368, -1.63419771194458, -1.3049312829971313, 1.0147334337234497, -0.9336094260215759, 1.0767297744750977, 0.8424339294433594, -0.785117506980896, -1.1395004987716675, -0.1723351925611496, -1.6675608158111572, -1.2348406314849854, -0.7174457311630249, 0.9553810358047485, 0.4484078288078308, -1.550743579864502, -2.021486759185791, 0.8234588503837585, -0.24075621366500854, -1.231345772743225, -0.06817077845335007, 0.6003772616386414, -1.1422688961029053, -1.1059073209762573, 0.2755086421966553, 0.6136115789413452, 0.28083452582359314, 1.5083647966384888, 0.8127315640449524, 1.0477194786071777, -0.3986939787864685, -1.0410557985305786], [-0.5100918412208557, 0.6232295632362366, 1.8086851835250854, 0.058501169085502625, -0.30232101678848267, -0.48470038175582886, -0.1976652592420578, -0.15354984998703003, -1.2580602169036865, 0.3079884648323059, -0.7354320883750916, 1.2649877071380615, 0.8330971598625183, -0.11703111231327057, -0.2380637377500534, -0.03434395045042038, -1.3172415494918823, -1.3996386528015137, 0.09636420756578445, -1.3303399085998535, 2.157578229904175, 0.6618550419807434, 1.7335355281829834, 0.0639161765575409, 0.33747774362564087, -1.090152621269226, -1.674756407737732, 0.9152576327323914, 1.9170223474502563, -0.5251891016960144, 0.624060332775116, 1.8611258268356323, 1.8477801084518433, -0.5633484721183777, 1.4938558340072632, 0.40963107347488403, 0.1846439689397812, 0.8825927972793579, 0.14180110394954681, -1.6132296323776245, 0.5788057446479797, 0.027845382690429688, 0.24108076095581055, -0.9677678346633911, -0.0036369096487760544, 1.2421290874481201, -0.5571635365486145, -0.3537626564502716, 0.3417311906814575, 0.889712929725647, 0.876240611076355, 0.40528053045272827, 0.7705699801445007, -0.8169817328453064, 1.4448463916778564, -0.3113642632961273, -1.217882513999939, -0.19266735017299652, -0.6672942042350769, 0.22888197004795074, 0.015578516758978367, -0.6969597339630127, 0.48042187094688416, 0.24925589561462402, 1.602264642715454, 0.27160996198654175, 0.4291018545627594, -1.6239728927612305, -1.0845803022384644, 0.8300182223320007, 0.4208700954914093, -1.2491097450256348, -1.5166501998901367, 0.5675206780433655, 0.16243787109851837, -0.3032681345939636, 0.31181979179382324, -0.9413207769393921, 0.48224684596061707, -0.26577889919281006, 2.314321279525757, -0.07155791670084, 0.719210684299469, -0.916061282157898, 0.49141359329223633, -0.0997769683599472, 0.68853759765625, -0.6569565534591675, -1.3604761362075806, 0.5333590507507324, -1.0769362449645996, -0.16720744967460632, 0.9936308264732361, 0.08390196412801743, -0.3452625274658203, -0.2585723102092743, 1.688887119293213, 0.404936283826828, -0.2224532514810562, 0.11664694547653198, 1.193724274635315, -0.845902144908905, 0.7335996627807617, -1.3365790843963623, -1.1822830438613892, 0.5146400928497314, -0.5787268280982971, 0.7504107356071472, -0.4608525037765503, 0.137781023979187, 1.0588897466659546, -1.3820810317993164, 0.4818870723247528, 0.9554759860038757, 0.2660694420337677, 1.5634238719940186, 1.8406100273132324, 0.3103412091732025, 0.40955406427383423, -1.7485411167144775, -2.490184783935547, -0.4006803333759308, 1.8000091314315796, -1.8723863363265991, -0.37754878401756287, 1.3213671445846558, 0.007028850261121988, -0.46481066942214966], [-0.32186463475227356, -0.03967634215950966, 0.2687079906463623, 0.9532909989356995, 1.8114705085754395, -1.734514594078064, 2.271930694580078, 0.06984341889619827, 0.5138711929321289, 0.08573366701602936, 0.09295016527175903, 0.6472401022911072, 0.6489006876945496, 0.681633472442627, 1.5458064079284668, -0.0970909371972084, -0.08495450019836426, 1.814531683921814, -1.7926243543624878, 0.39246806502342224, 0.6539990305900574, -0.30611011385917664, -0.35094743967056274, -0.9589783549308777, 0.6857471466064453, -0.927910566329956, -0.6147103309631348, 0.5748581886291504, -0.458061158657074, 0.7877654433250427, -0.5384759306907654, -1.1874891519546509, 0.4493350684642792, 0.6752777695655823, 0.5707970857620239, -1.4812825918197632, -0.08675703406333923, -0.9255431294441223, -0.2040042132139206, 0.18977896869182587, -1.5580581426620483, -1.841045618057251, 0.21348842978477478, 1.0965899229049683, 1.5269832611083984, -0.11534105986356735, 0.971604585647583, -0.5265868902206421, 0.7122147679328918, 1.0948179960250854, 0.4449608325958252, 1.4941643476486206, 0.7084054946899414, -0.45236706733703613, 1.0605899095535278, -1.3721188306808472, -0.7546606063842773, -0.28813499212265015, 1.6900383234024048, -0.3709995150566101, -0.28127819299697876, -0.9919174313545227, -1.5861835479736328, -0.17898234724998474, 1.5788121223449707, 0.6694008111953735, -0.1745292693376541, 0.38342100381851196, 0.5415528416633606, -0.4322778880596161, 0.7675319314002991, 0.6963223218917847, -0.2946287989616394, 0.8865789175033569, -0.38086870312690735, 1.7134145498275757, 0.4869231581687927, -0.701119065284729, -0.3124637305736542, -0.6361902952194214, -0.8836224675178528, -0.25512775778770447, -0.7839655876159668, 0.9924608469009399, 1.7210819721221924, 0.3123651146888733, 1.613970398902893, -0.956985592842102, -0.5637637972831726, 1.3757643699645996, -0.18901710212230682, 1.8050817251205444, -0.6918593645095825, 0.3659283220767975, -1.3524326086044312, 0.7275193333625793, 0.33131372928619385, 1.201829195022583, -0.23585999011993408, 1.8236713409423828, 1.3162460327148438, 0.029693076387047768, 0.6687415242195129, -0.24559548497200012, 0.7651791572570801, -0.9464155435562134, 0.0265171118080616, 2.7770628929138184, 1.464986801147461, -1.3254687786102295, -0.6006187796592712, 1.1997535228729248, 0.013212078250944614, -0.1837039291858673, -0.7462632060050964, 0.45851871371269226, -0.016358910128474236, -0.21670542657375336, 1.2141326665878296, 0.3329261541366577, -2.2432668209075928, -0.3270530700683594, -1.0930564403533936, 0.2543773651123047, 0.3539987802505493, 0.012547356076538563, -0.16854654252529144, 0.2511204183101654], [-0.08623301237821579, -0.7500137686729431, -0.0163438618183136, 0.2569428086280823, -0.9930922985076904, 1.041405439376831, -0.33061373233795166, 0.41478976607322693, 1.2213695049285889, 0.5913141965866089, -1.4720873832702637, 0.028427133336663246, 0.14660370349884033, -0.5930120944976807, -1.216372013092041, -0.21267813444137573, 2.1859066486358643, -0.16824203729629517, 0.9048829078674316, 0.5533968210220337, -1.1170618534088135, -0.8189292550086975, -0.47958359122276306, 0.15114712715148926, -0.053280167281627655, -0.7524537444114685, -0.20233575999736786, 0.6861377358436584, 1.4037141799926758, -0.08263858407735825, 0.16898193955421448, -1.0891107320785522, -0.6220728754997253, -1.5963473320007324, 1.2077999114990234, -1.3852849006652832, 0.35383254289627075, 0.6661660075187683, -1.3632482290267944, -0.1844484955072403, -1.2887412309646606, 0.8529197573661804, -1.0487139225006104, -0.4399177134037018, 0.36342063546180725, -0.02047405205667019, -0.5313677191734314, -1.0974773168563843, -1.3010878562927246, 0.18668042123317719, 0.064715176820755, -1.6962802410125732, -0.3133244812488556, 0.6063781380653381, 1.4275840520858765, 0.6745781898498535, -0.2623496651649475, -0.13037893176078796, -0.6610027551651001, -2.2970175743103027, -0.34139761328697205, 0.46033549308776855, 1.4946037530899048, -0.051780279725790024, -1.4767026901245117, -0.9026824235916138, 0.6896226406097412, 0.3618784248828888, 0.8117616772651672, 0.6553581357002258, 0.3668031394481659, 0.6174401044845581, -1.882096767425537, -0.21411144733428955, 0.25653940439224243, 0.025592559948563576, -0.7930683493614197, 2.2063491344451904, -1.0505633354187012, -1.073182463645935, 0.2952166497707367, -1.490604281425476, 1.5861715078353882, 0.16269521415233612, -1.7811853885650635, -2.182309627532959, 0.728000819683075, -0.38749220967292786, -0.9117146134376526, -2.7906105518341064, 1.744601845741272, -1.3145548105239868, -0.38897252082824707, -1.0951672792434692, 0.5989730358123779, -1.6974537372589111, 0.9237443208694458, 0.29999998211860657, -0.05933864042162895, -1.6104655265808105, 1.176109790802002, -1.1296696662902832, 0.6842830181121826, 0.22723886370658875, 0.10922832041978836, -1.577289342880249, -1.0140552520751953, 0.8389034271240234, 0.9061703681945801, 0.14515255391597748, 1.025789499282837, 0.3279099464416504, -1.2837504148483276, -0.46282505989074707, -0.08283045887947083, 1.1198683977127075, 0.91656494140625, -0.6921628713607788, 0.6225461363792419, 0.905899703502655, 1.0117751359939575, -0.32295775413513184, 0.6832431554794312, -2.2285263538360596, -0.17586474120616913, 1.1382993459701538, -0.760958731174469, -0.029910411685705185], [0.7508143782615662, 0.8804890513420105, 0.6675572395324707, 1.2077977657318115, 0.4600943624973297, 0.9378392696380615, -1.9473812580108643, -1.1537322998046875, -0.16474321484565735, 1.1277580261230469, 2.4442431926727295, -0.9367775321006775, -0.9721737504005432, -0.4643744230270386, -0.619594931602478, -0.7061487436294556, -0.15942442417144775, -0.8670228719711304, 0.01695151813328266, -1.510236382484436, -0.18283408880233765, 1.3224706649780273, -0.245241180062294, -0.7640751004219055, -0.47562775015830994, 1.4019283056259155, -0.053708963096141815, -2.2001841068267822, -0.6359044909477234, 0.044209886342287064, 1.811004877090454, -0.9908022880554199, 0.3538617789745331, 1.327064037322998, 0.056219737976789474, 0.27827394008636475, -0.3237343430519104, -0.3963966965675354, 0.9232738614082336, -0.27375495433807373, 0.6500082612037659, -0.4978249669075012, -0.3089657127857208, 0.5357304811477661, -0.5490400791168213, 0.4315902888774872, -1.9481275081634521, -0.8887869119644165, -1.7243211269378662, 0.07717035710811615, -0.09774759411811829, -0.5544987916946411, 0.6591371297836304, 0.17157045006752014, 0.8075145483016968, 0.8738911747932434, 0.6165876388549805, 1.4331296682357788, 1.6303634643554688, -0.9878137707710266, 0.07394150644540787, -0.25234997272491455, 1.0088897943496704, 1.024867057800293, 0.6765250563621521, 1.6195272207260132, 0.8306647539138794, -0.07609399408102036, 0.9082561731338501, -2.1670937538146973, -0.45381927490234375, 0.4640326499938965, 1.0024744272232056, -0.21473892033100128, -0.4924451410770416, -0.4820663034915924, -1.4255971908569336, -1.2597631216049194, -0.6495846509933472, -1.4964900016784668, -0.4704555571079254, 0.8134022951126099, -1.7512692213058472, 2.165767192840576, -0.7120457291603088, 2.561640501022339, -0.6157405376434326, -0.0510292686522007, 0.36992284655570984, -0.8233150243759155, -1.0189552307128906, -0.7379257082939148, 1.9517027139663696, 0.7233820557594299, 0.6804953813552856, 1.6275087594985962, -0.7661633491516113, 0.8695118427276611, -1.3984888792037964, -0.4666454792022705, -0.9323054552078247, -0.7259492874145508, 0.27547773718833923, -1.077200174331665, -0.4127342700958252, 1.995727300643921, -1.7663202285766602, 0.820022702217102, -1.7676922082901, 0.867838442325592, 0.7637401819229126, -0.6747188568115234, 0.35135889053344727, -0.18159155547618866, -1.3461503982543945, 0.5745491981506348, -2.842054605484009, 0.1438969075679779, -1.952036738395691, 0.7283703684806824, -0.45431262254714966, -1.3630340099334717, -0.3836321532726288, 0.27062252163887024, -0.14044919610023499, 0.8798125982284546, 0.043228425085544586, -0.35047706961631775], [0.09243419766426086, -0.19719786942005157, 0.35526973009109497, 0.2903487980365753, 0.5528693199157715, -0.6956190466880798, 0.2504681944847107, -0.05103857442736626, -1.3633816242218018, -1.2183830738067627, 0.6730000972747803, -0.9549188613891602, 0.5841822028160095, -0.6956800222396851, 0.17459756135940552, 0.717405378818512, -0.746630072593689, 1.265425682067871, -0.3684048354625702, -0.40494999289512634, -1.1012694835662842, 0.08781357109546661, -1.2683628797531128, 0.4143211245536804, 1.0672417879104614, -0.033794231712818146, 0.2660815417766571, -0.46429818868637085, 0.033290181308984756, 1.3110535144805908, 0.3958507180213928, 0.9012513160705566, -0.6596696972846985, 1.2469475269317627, -1.0979597568511963, -0.5866738557815552, -0.908618152141571, 1.895112156867981, 1.4237335920333862, -0.41727033257484436, -0.2879008948802948, 0.08141497522592545, 1.860302448272705, 2.1016297340393066, 0.7336816787719727, -1.0851929187774658, 2.319361686706543, 0.2436305284500122, 0.11921555548906326, -0.11179198324680328, -0.5629292130470276, -1.2131465673446655, -0.8960766196250916, 0.17367786169052124, 1.1985284090042114, -1.676523208618164, 0.5584713816642761, 0.5194028615951538, 0.3474707305431366, 1.1561297178268433, 0.7662968039512634, 1.1111806631088257, 1.0788037776947021, 0.6233105659484863, -0.3051409423351288, -0.8988713622093201, 0.24259339272975922, 1.6303383111953735, 0.6146303415298462, 1.5490567684173584, 0.9459452033042908, 0.6571423411369324, -1.3454793691635132, 0.8886927962303162, -0.5446719527244568, -0.38791152834892273, -2.4319112300872803, -0.40773236751556396, -1.1748136281967163, 0.2858582139015198, -0.17120547592639923, -0.0790870264172554, 0.15939827263355255, 0.10829578340053558, -1.1894711256027222, -0.2978963255882263, 0.6899535655975342, -1.0167099237442017, -0.7325149774551392, 0.11312716454267502, -0.20729592442512512, 0.002055802382528782, -0.4838242828845978, 1.5647599697113037, -2.0796759128570557, 0.7125909328460693, -0.1912710815668106, 0.6761711239814758, -0.20555053651332855, -0.11007510870695114, 0.7637658715248108, -0.3933955132961273, -1.686410903930664, 1.292258858680725, 1.0592433214187622, 1.2743661403656006, -2.1640541553497314, 0.4489961564540863, -0.475034236907959, -0.7244725227355957, 1.2118444442749023, -1.178162932395935, 1.4564638137817383, -1.3166654109954834, -1.0927317142486572, -0.850770115852356, 2.2462801933288574, 0.3514675796031952, -1.0748573541641235, 1.156882405281067, -0.25487250089645386, 1.7231451272964478, 0.4991452097892761, -1.4501631259918213, 0.8187306523323059, -0.6427155137062073, 0.1880887895822525, -1.6650077104568481], [0.6861661672592163, 1.1586965322494507, -0.9540385603904724, -0.1030685305595398, 1.808287501335144, -0.3017609417438507, -0.2780129015445709, 0.5054608583450317, 0.3637795150279999, -0.7000718116760254, -1.3082404136657715, -1.3744256496429443, 0.4589519798755646, -1.1561691761016846, -2.1790332794189453, -0.11086831241846085, 0.14987094700336456, 0.9963992834091187, 1.4258062839508057, 0.7011227607727051, 1.310156226158142, -0.7431901693344116, -1.2558366060256958, 0.023980015888810158, -0.834385335445404, -0.3425386846065521, -0.37717390060424805, -0.5936740040779114, 0.48807549476623535, 0.9901614785194397, 0.06557508558034897, -1.3909579515457153, 0.28357672691345215, 0.38667401671409607, 0.09997442364692688, -0.5870243906974792, 1.423701524734497, 0.6191157102584839, 0.001158414175733924, 0.07595094293355942, -1.1847572326660156, -0.831809401512146, 0.3220081329345703, 0.07360544055700302, -0.9255780577659607, 0.17337119579315186, -0.112961046397686, 0.13435505330562592, 1.9877710342407227, 0.3874928653240204, -0.720302402973175, -1.1336402893066406, 0.6234480738639832, 1.3270162343978882, 0.7199123501777649, -1.7548953294754028, 1.5833368301391602, 2.444082021713257, 0.2530279755592346, -0.5966771245002747, -0.7220675349235535, 0.2278953343629837, 1.4989172220230103, 1.1962777376174927, 1.7826393842697144, 0.13600073754787445, -1.3209055662155151, 1.5678257942199707, -2.5782148838043213, -1.0312340259552002, 0.8931007385253906, -2.0318009853363037, -1.6357457637786865, -0.5567817687988281, -1.4498664140701294, 1.7784401178359985, -0.4506191909313202, 1.6513590812683105, 0.13584622740745544, -0.47538861632347107, -0.06190406531095505, 0.2965027093887329, -0.3547075688838959, 1.2090433835983276, -0.32656362652778625, -0.8842399716377258, -1.2870761156082153, -0.8161007165908813, 0.13445818424224854, 1.7426650524139404, 0.5717208385467529, 0.3058224022388458, -0.2291405349969864, 0.11176743358373642, -0.2636219263076782, 0.39001065492630005, 1.2446072101593018, -1.3898266553878784, -2.3219974040985107, -0.13376645743846893, 1.158827543258667, -1.0453041791915894, 0.9944304823875427, 0.8040704131126404, 1.0886515378952026, -1.4835057258605957, -0.29238077998161316, 0.0662156417965889, -0.20581039786338806, -1.2875442504882812, -0.6347774863243103, -0.1514122635126114, -0.32482677698135376, -0.3229949176311493, 0.3071039319038391, -0.3714844882488251, -0.9206189513206482, -0.18909025192260742, -0.902641236782074, -0.9475052952766418, 0.5181182026863098, -0.18640758097171783, 0.4773140847682953, -2.128268241882324, -1.649412989616394, 0.34098920226097107, -0.6843554973602295, -0.5793932676315308], [-1.4107974767684937, 1.164811372756958, 0.7825561165809631, 0.46355417370796204, 0.07010312378406525, 0.8636038899421692, 1.4895273447036743, -1.1994339227676392, 1.456946611404419, -1.2536201477050781, -0.9576086401939392, -1.507629156112671, -0.7479499578475952, -1.060295581817627, 0.9322208166122437, 0.44512873888015747, 1.8490116596221924, 0.04864639416337013, 0.5610709190368652, 0.9775897860527039, -0.1915052831172943, 0.8026541471481323, 0.6198790073394775, -0.5108618140220642, 1.4570748805999756, 2.3536465167999268, -0.0713043063879013, 0.1421128362417221, 0.4584019184112549, 0.10341149568557739, 0.4548494517803192, 0.9704174995422363, 1.008022665977478, -1.49393892288208, -0.6404075622558594, -0.06996802240610123, -0.5878878235816956, 0.7541680932044983, -0.28572705388069153, 0.08310143649578094, 0.10198412090539932, -1.4311068058013916, 0.16659654676914215, 0.4616917669773102, -0.28317826986312866, 2.661534547805786, -1.7532397508621216, -0.018531406298279762, 1.03917396068573, 0.4408700466156006, -0.15750063955783844, 1.0936970710754395, 1.0546640157699585, -0.5418999195098877, 1.8543590307235718, -0.572762131690979, 0.5438879728317261, -0.04090338200330734, 1.5397543907165527, 2.3695735931396484, 1.4170085191726685, -1.197073221206665, 1.37533700466156, -0.6519966721534729, -0.2713627219200134, -0.949390172958374, 1.500816822052002, -0.6088547110557556, 1.5969139337539673, -0.27027103304862976, -1.850471019744873, -0.49589410424232483, -1.1700470447540283, 2.2715036869049072, -1.314210057258606, 0.8080595135688782, 1.006608247756958, -0.5224975943565369, -0.014940634369850159, -0.7134578227996826, 0.18429236114025116, 0.355689138174057, -1.146065592765808, -1.8783930540084839, 0.1423303335905075, -0.3044191002845764, 0.35446953773498535, -0.2886090874671936, 1.0927938222885132, -0.9498319625854492, 0.3493765592575073, -0.16784512996673584, 0.2237645536661148, 0.2396814078092575, -0.7843520045280457, 0.4843965172767639, 2.8160958290100098, 0.16600815951824188, 0.16137798130512238, 3.119450569152832, 0.8604443073272705, -1.633481502532959, -1.3623262643814087, 0.5767697691917419, 1.7226760387420654, 0.29744774103164673, -1.2431000471115112, 0.6141732335090637, -1.0467907190322876, 0.8161552548408508, 0.8506627082824707, -0.2407366931438446, -1.0049684047698975, -1.299586296081543, -1.1480224132537842, 0.8401852250099182, -0.7174710035324097, -0.35106512904167175, -1.671422004699707, 0.4103314280509949, 2.6671302318573, 0.49758535623550415, -0.18691237270832062, 0.30552029609680176, -1.0337241888046265, -0.07174956798553467, -0.6159740090370178, 0.48334774374961853], [0.3120023012161255, 0.2064107060432434, 0.29493582248687744, -0.2830115258693695, 0.8840492963790894, 0.753923237323761, -0.595608651638031, 0.03425909951329231, -0.2604927122592926, 0.9612552523612976, 1.8317197561264038, -1.2351499795913696, -0.7543461918830872, 0.4274824261665344, -0.7655925154685974, 1.5265636444091797, -2.6210954189300537, -0.8450833559036255, -0.8019245862960815, 0.2128680944442749, 1.2948946952819824, 0.855659544467926, -2.3721747398376465, 2.0434763431549072, 0.9966643452644348, 0.025749558582901955, 1.3812825679779053, 0.06953930109739304, -0.5126495361328125, 1.2516350746154785, 0.14927557110786438, 0.8419896364212036, -0.5065218806266785, -0.12951351702213287, -0.7204043865203857, 0.7494507431983948, 0.22513625025749207, 1.140165090560913, 0.1405654400587082, 1.4386134147644043, 0.09486936032772064, 1.6417573690414429, 0.2624863386154175, 0.20023934543132782, 0.606215238571167, 0.5001251101493835, -0.03658226504921913, 0.9809300899505615, 0.5470807552337646, 0.2765684723854065, -0.35999795794487, 0.5465337634086609, -1.3668169975280762, 0.3330184519290924, 2.1022274494171143, -1.0130921602249146, -0.16304410994052887, 2.189793825149536, -0.24048694968223572, 1.6812217235565186, 0.8208482265472412, 0.8831862211227417, -0.4664374887943268, -1.0357345342636108, -0.6045982837677002, 0.7480955719947815, -0.42538729310035706, 0.7345123887062073, -1.0520159006118774, 0.17982704937458038, -0.09401337057352066, 0.7049108743667603, -0.6491562724113464, -1.1253000497817993, 0.3104458749294281, -0.43350112438201904, 0.9214242696762085, 1.8117469549179077, -0.18384012579917908, -0.03366899490356445, -1.429124116897583, -0.23894871771335602, 0.7363168597221375, 0.7775614857673645, 0.34998640418052673, 0.7101022005081177, -1.3948842287063599, 1.977696418762207, 0.2538602650165558, 0.4166052043437958, 1.3424550294876099, 0.1385132372379303, -1.2144153118133545, -1.1494771242141724, 0.36458534002304077, 0.642536997795105, -0.764325737953186, -0.19508102536201477, 0.39220571517944336, 0.04149281978607178, -0.176932230591774, -0.4349747598171234, 0.14064154028892517, 1.2106163501739502, 0.44444870948791504, 1.5961788892745972, 0.12734083831310272, 0.9582749605178833, -0.16429343819618225, -0.27679699659347534, 1.0377057790756226, -0.1590445190668106, -0.7002177834510803, 0.2969692349433899, -1.2177362442016602, -0.425715833902359, 0.5435605049133301, 1.606937050819397, -0.24453826248645782, -0.6394344568252563, -1.3051056861877441, -0.5060433149337769, 0.7018314599990845, 0.8343087434768677, -0.3598111569881439, -0.3011554181575775, -0.803534209728241, -0.03841876611113548], [-1.786266565322876, 0.9422690272331238, 0.2611129581928253, 0.6461262702941895, -0.12669146060943604, 1.872568964958191, 1.181038498878479, -0.39825621247291565, -0.26362836360931396, 0.12090317159891129, -0.2831224501132965, -0.05526662990450859, 0.8248088359832764, -0.02585688605904579, -0.22020787000656128, -0.15157467126846313, -0.35861366987228394, -0.6345399618148804, 0.48920825123786926, -0.19624224305152893, -0.09130654484033585, -0.6993352770805359, 0.06492653489112854, -0.9411167502403259, -0.2739810645580292, -2.106375217437744, 0.7924509048461914, -1.8782037496566772, 0.6039733290672302, 0.8149853348731995, 0.9372628331184387, -0.8463279008865356, -1.9299486875534058, -0.9522243142127991, -2.0212621688842773, -0.9684590697288513, -0.1959788203239441, 0.1452917754650116, -0.5905547142028809, -0.09878353029489517, 0.6940068602561951, -1.3569960594177246, 1.2762155532836914, 1.9657635688781738, 1.5696275234222412, -0.5842763781547546, 0.6829372048377991, 0.10567888617515564, -0.7485142350196838, 1.4209479093551636, -1.6028305292129517, -0.6733203530311584, -1.2114460468292236, -0.27000319957733154, -1.3823527097702026, -1.1288121938705444, 2.08378005027771, 0.6811627745628357, 0.5481606125831604, 0.16044622659683228, 0.15690232813358307, 0.04397150129079819, -0.813666582107544, 1.418843388557434, -0.593859851360321, -1.933053731918335, 0.7024792432785034, -0.8400307297706604, 0.6020234823226929, 1.3559948205947876, -0.09583178907632828, 0.38540366291999817, -0.6242464184761047, -0.10865715146064758, 0.18497389554977417, 1.3539108037948608, -1.0874825716018677, 0.9859365820884705, 0.5471537709236145, 0.37114012241363525, -0.8034869432449341, 1.0677852630615234, -0.9391744136810303, 0.42515406012535095, -0.8655396103858948, 1.7993487119674683, 0.04785827919840813, -0.029730115085840225, -2.146721839904785, 1.3640713691711426, 0.33150655031204224, 1.1739308834075928, 0.520731508731842, -0.5572770833969116, -1.6464297771453857, 0.01840073987841606, 1.8839735984802246, 0.3009081780910492, -1.8000757694244385, 1.695286750793457, 0.752435028553009, -0.25293949246406555, -1.3529070615768433, 0.8457263112068176, -1.1691657304763794, 0.04808575287461281, -1.1345889568328857, 0.6738744378089905, 0.4907242953777313, 0.5237848162651062, 0.44562551379203796, 0.07224800437688828, -0.8609312176704407, -0.333584189414978, 0.07678785175085068, 0.621846616268158, 0.26409822702407837, 0.7978019118309021, 0.5159680843353271, -1.0419379472732544, -0.07952912896871567, -1.8178967237472534, 1.6402946710586548, 0.12395127862691879, 0.30017006397247314, -3.191709041595459, -0.0181479062885046, -1.0754257440567017], [0.672335684299469, -0.04377516731619835, -0.34329742193222046, -0.3072390556335449, 0.09225436300039291, 0.5074530243873596, -0.3907258212566376, 0.25382986664772034, -0.3730001747608185, 2.1949973106384277, -0.819499671459198, -1.7539777755737305, -1.3954998254776, 2.0775957107543945, 0.7154034972190857, 0.6953958868980408, 0.9925193190574646, 0.4601011872291565, 0.2177465409040451, -1.5876617431640625, -1.6456406116485596, -1.609087347984314, -0.24238064885139465, -1.3544007539749146, -0.8736339807510376, -0.3910912871360779, -1.9267492294311523, -0.5628005266189575, 1.715964913368225, -0.07891280204057693, 0.18601514399051666, -0.7478120923042297, -0.7459470629692078, -0.9567577838897705, 1.6340951919555664, -0.2509170472621918, 0.8623403310775757, 1.8004413843154907, -0.45957034826278687, 0.3973775804042816, 0.3614029586315155, -1.0425692796707153, -0.4164256751537323, 1.4014471769332886, 0.666182816028595, 0.19344884157180786, 1.0144710540771484, -0.37174832820892334, 0.8597614169120789, -0.7369722723960876, -0.5408207178115845, 0.9447195529937744, 1.2774417400360107, 0.06515564769506454, -0.46176984906196594, -1.7491803169250488, 0.5269985198974609, -1.2226873636245728, -0.3106762766838074, 1.0215232372283936, 0.9906831979751587, -0.3385036289691925, -0.6592729687690735, -0.2666237950325012, -1.0514658689498901, 1.7141547203063965, -0.47334524989128113, -0.9010960459709167, 1.2377876043319702, 0.2921079695224762, -0.23698623478412628, 1.0572054386138916, -2.2796530723571777, -1.7405815124511719, -0.07410617917776108, 0.4980258047580719, -0.47794243693351746, 0.4893878102302551, -0.2935083508491516, -2.6959989070892334, -1.6009671688079834, -1.5643786191940308, 0.1740335077047348, -0.37354007363319397, -0.23932628333568573, -0.755660891532898, 1.3618770837783813, 0.6806793212890625, 0.4570806920528412, -0.6017734408378601, -1.001293659210205, -0.3380434811115265, 0.6417784094810486, 0.4487229585647583, 0.5366929769515991, 0.07766091823577881, 0.043082885444164276, 2.5309550762176514, -1.4203678369522095, 0.7315963506698608, 0.8892641663551331, -0.16202369332313538, -0.28723058104515076, -0.908298134803772, 2.2363548278808594, 0.38736504316329956, 0.8021628260612488, -0.3467722237110138, -1.2663459777832031, 1.3242645263671875, -1.6417566537857056, 0.9983874559402466, -2.605699300765991, -0.39235395193099976, -0.5969581604003906, -0.5757119655609131, 0.4330138564109802, 0.9880099296569824, -0.030738120898604393, 0.05619833990931511, 0.3564167618751526, 0.3994365930557251, -1.1573485136032104, 0.294023722410202, -2.5547072887420654, 2.3312201499938965, 0.29071348905563354, -0.6317571401596069], [0.3920152485370636, 0.08491010963916779, 0.14989131689071655, -0.11895479261875153, -0.7717357873916626, -1.6934568881988525, -1.3371453285217285, -0.1053435355424881, 0.815041720867157, 0.6734583377838135, -0.7124800086021423, 0.18746784329414368, 1.8948349952697754, -0.9340279698371887, 0.03898165002465248, -0.5244053602218628, -1.7038135528564453, 0.5558773279190063, 0.8169310092926025, 0.4527648687362671, -1.6455756425857544, -1.0116984844207764, 0.40506768226623535, -0.7635840177536011, 0.33487385511398315, 0.7284679412841797, -0.5479484796524048, 0.464743971824646, 0.9272478818893433, -0.35394221544265747, 0.48869025707244873, 0.9053283333778381, 0.6641244888305664, 0.773732602596283, -0.658649206161499, 1.677804708480835, 0.499105840921402, 0.6513407230377197, -0.3214348256587982, 0.5161388516426086, -0.6087324023246765, -0.5731064677238464, -0.5818008184432983, -0.045008644461631775, -0.5633498430252075, -0.4056597948074341, -0.17718903720378876, 1.1389483213424683, 1.4073301553726196, 0.46625038981437683, 0.17777760326862335, -0.7416609525680542, 0.9140459895133972, -0.5620803833007812, 0.07321354746818542, 0.7490499019622803, -0.5689310431480408, 0.8644686341285706, 0.6213691234588623, 0.38813695311546326, 0.6906439661979675, -0.15174216032028198, 0.5476974844932556, -0.5393266677856445, 2.2801778316497803, 1.0786256790161133, -0.3365044593811035, 0.3015473783016205, -1.4294527769088745, 0.8630412817001343, -1.5183782577514648, 1.1557773351669312, -0.2761836349964142, 0.13596922159194946, 0.29351767897605896, -0.25469473004341125, -0.050514236092567444, 0.20727108418941498, -0.3129894435405731, 0.5134942531585693, -0.02470874786376953, 0.47740480303764343, 1.243359923362732, 0.6876233220100403, 0.8363791108131409, 0.3961212933063507, -1.3232296705245972, 0.12490662932395935, 0.9601711630821228, -0.7152007222175598, -0.5097843408584595, -1.1034237146377563, 0.983727753162384, 0.673211395740509, 0.1256844401359558, -0.833860456943512, 0.558692455291748, 0.2794424593448639, 0.40488266944885254, 1.3281618356704712, -0.2055538296699524, -0.8577921986579895, 0.6314336061477661, 0.09789128601551056, 0.2208210825920105, -0.8639317154884338, 0.01965269446372986, 0.3789984881877899, 0.9819303154945374, -0.3970579504966736, -1.3423810005187988, 0.32324856519699097, -0.04873540252447128, 0.6514636278152466, -0.7363355159759521, -0.0030362228862941265, -0.2723965644836426, -0.5520656704902649, -0.3504524230957031, 0.4232366979122162, -0.316232293844223, 0.7353672385215759, 0.5759274959564209, -0.8014507293701172, 0.30317384004592896, 2.3202755451202393, 0.5876634120941162, -1.725929856300354], [0.8026155829429626, 0.7102307677268982, 0.6451478004455566, -0.269612193107605, 0.6163864731788635, 0.355998158454895, -2.6578426361083984, -1.0780940055847168, -0.3769061267375946, 2.138927459716797, 0.8173722624778748, -1.4889287948608398, 1.385215163230896, -0.14323526620864868, 1.6279845237731934, -0.9976480007171631, -0.040858589112758636, 1.9347950220108032, -0.17653918266296387, 0.6461256146430969, -0.5010639429092407, 0.911249577999115, 1.8683793544769287, 0.9918854236602783, 0.6623285412788391, 1.804895281791687, 0.8327739238739014, 1.9159841537475586, -0.26195982098579407, 0.37040916085243225, -0.35857924818992615, -0.2611192464828491, 1.681963562965393, -0.7159780859947205, -0.3880980610847473, 0.5034023523330688, -1.2792564630508423, 1.6590195894241333, 0.3516051173210144, -0.537039041519165, 0.7093722820281982, 1.137725830078125, 1.353785514831543, -0.3038044571876526, -1.1314080953598022, 0.24301457405090332, 1.0896245241165161, 0.5400462746620178, 1.2623318433761597, -0.38871604204177856, 0.5878885984420776, 1.2781071662902832, -1.2853020429611206, -1.229544997215271, -0.37801486253738403, -0.095171257853508, -1.680314302444458, -1.385546326637268, -0.4853668808937073, -0.2759729325771332, -0.6753403544425964, -0.6249889135360718, 0.7792966365814209, 2.024217367172241, -0.6806603074073792, -0.10884439945220947, -1.6084849834442139, 1.4775526523590088, 1.0206609964370728, 1.3510651588439941, 2.4148964881896973, -0.5493259429931641, -1.0984874963760376, -0.1958955079317093, -0.16515061259269714, -1.1688604354858398, 1.3008403778076172, -1.9475098848342896, -1.3142071962356567, -0.7805539965629578, 1.1081650257110596, 0.4296443462371826, 0.20396867394447327, 0.8359545469284058, 0.1552858203649521, 0.07126142829656601, 0.8623380661010742, 1.8630807399749756, 0.45474672317504883, 0.3995457887649536, -0.3611268103122711, -0.15570613741874695, 0.14647255837917328, -0.10410735011100769, -0.5608302354812622, -0.999806821346283, -0.14178423583507538, 0.2820449471473694, 1.1025285720825195, 2.820695400238037, 0.4976065456867218, -0.4603511393070221, 0.24798700213432312, -0.7869279384613037, 1.2128130197525024, 0.9595357179641724, -0.45064765214920044, 1.0494542121887207, 0.5033331513404846, 2.0010716915130615, -0.9481068253517151, -1.666926622390747, -1.7203036546707153, -0.5586159825325012, 1.010411262512207, 1.3870457410812378, 1.3481794595718384, -0.028919987380504608, -0.36733511090278625, -0.03217808157205582, 0.1995086967945099, 1.0614835023880005, -0.522196888923645, -0.6583558320999146, 0.0671318843960762, -0.5341055989265442, 1.3377082347869873, -1.7413909435272217], [0.21310612559318542, 0.24511957168579102, 1.6602405309677124, -0.23770171403884888, 0.2953520119190216, 0.30285996198654175, 0.007288586813956499, -0.25855275988578796, -2.235724687576294, -1.2658138275146484, 0.6045657396316528, -1.692233681678772, -0.7897747159004211, 0.7223923206329346, -0.890565812587738, 0.5870625376701355, 1.9768625497817993, 1.182778239250183, -0.23985205590724945, 0.0364183746278286, 0.8942501544952393, -0.4716412425041199, 0.8127124905586243, 0.04698106646537781, 1.0585051774978638, 0.30137279629707336, -0.32336482405662537, -0.4503672122955322, -1.2936137914657593, 0.563993513584137, -0.2870418131351471, -0.7125111818313599, 0.7234960794448853, -0.012375996448099613, 0.11932238936424255, -0.044548485428094864, 0.6158579587936401, 0.3907325565814972, -1.0968050956726074, -0.42137864232063293, 0.03200346231460571, 0.05001510679721832, -2.042642593383789, 0.6204112768173218, -1.2135919332504272, -0.6517060399055481, 0.42395442724227905, -0.5133547186851501, 1.3606995344161987, 0.3809228539466858, -0.9497675895690918, 0.8430781960487366, 0.4640395939350128, -2.174170732498169, 0.7721874713897705, 0.2952338755130768, 0.22879596054553986, 0.4387497305870056, -0.1255018562078476, -2.2828094959259033, 0.5901933312416077, 1.1048871278762817, -0.5769066214561462, 1.1111774444580078, -0.46426230669021606, -0.9557783007621765, 1.042396068572998, -1.3467249870300293, 0.8819721937179565, -1.2602965831756592, -2.087491273880005, 1.3593240976333618, -0.32991132140159607, 0.8121670484542847, 1.729248285293579, 0.3591395914554596, 1.34725821018219, -0.19903966784477234, 0.29238462448120117, 2.301600217819214, 0.25358110666275024, 1.1313505172729492, -0.8588704466819763, -0.8368743062019348, 0.9337127804756165, 0.34660404920578003, 0.7444605827331543, -2.07725191116333, 1.5785155296325684, -0.424725741147995, -1.1830655336380005, -0.49338820576667786, 0.316983163356781, -0.9872938394546509, 0.31020480394363403, -0.05329159274697304, 0.4692935347557068, 2.359999895095825, 1.0440946817398071, -2.2010467052459717, -1.0425511598587036, 0.3925487995147705, 0.6616230607032776, 2.5925819873809814, -0.23777036368846893, 1.6610931158065796, -0.3243774473667145, 2.4044182300567627, -1.3243662118911743, 0.7871351838111877, 1.367166519165039, 0.8384739756584167, 0.512917160987854, 0.32124316692352295, 0.9523928165435791, 0.7879519462585449, 0.8989421129226685, 0.30290165543556213, -0.17772932350635529, -0.9071411490440369, -0.14820168912410736, 0.10836759954690933, 0.12474188953638077, -1.4332051277160645, -2.295827627182007, -1.3624955415725708, -0.1644139140844345, -1.1522271633148193], [-1.637986660003662, 0.738125741481781, -0.6774051785469055, 0.4592491388320923, -0.3715366721153259, -1.0962026119232178, 0.08246994018554688, 1.1769804954528809, 0.2436036765575409, -0.03461221233010292, -1.5961546897888184, 0.6211991310119629, 2.035123109817505, -0.3185581564903259, 0.5234798789024353, -1.4491125345230103, 0.6221354603767395, -2.742198944091797, 0.04780952259898186, 0.37679779529571533, -0.3518877923488617, 0.6941870450973511, 1.8256466388702393, 1.0720049142837524, -0.3690791130065918, 0.39026275277137756, -0.4770588278770447, 1.2869775295257568, 1.2869683504104614, 0.05974571034312248, -0.6363973021507263, -1.0147292613983154, 0.702061653137207, 0.7473598122596741, 1.8378326892852783, 1.713668942451477, -1.1267898082733154, 2.3497142791748047, 0.8091829419136047, 0.9588602185249329, 0.8245735764503479, 1.2262190580368042, 0.5595541596412659, -0.27721214294433594, 0.05685257911682129, -0.4079541563987732, 0.6447554230690002, -1.8132691383361816, -0.9985966086387634, -2.871609926223755, 1.5459057092666626, -0.15273001790046692, 0.45487335324287415, 0.837251603603363, -0.8185442686080933, 0.3074153959751129, 0.2352675199508667, -1.0358189344406128, -0.35353466868400574, 1.8720364570617676, 0.580535888671875, 0.4474434554576874, 0.4971832036972046, 0.4662570059299469, -0.5451416969299316, 0.14628763496875763, -0.4629269242286682, -1.033962607383728, 0.2873956561088562, -0.7335017919540405, 0.09637454152107239, -1.6332321166992188, -0.7917255759239197, -0.03842857480049133, -1.7739566564559937, -0.28692352771759033, -2.0166404247283936, 0.6727204918861389, 1.2765593528747559, -1.2189953327178955, 1.1449352502822876, -0.5360578894615173, -0.03919437155127525, 1.090104579925537, 0.2882288992404938, 1.315140962600708, -0.6365870833396912, 0.3516559302806854, -1.634277582168579, 2.1112542152404785, -0.6812499761581421, 0.873129665851593, -0.39998510479927063, 1.4079633951187134, 0.346800297498703, 0.011656287126243114, 0.6849363446235657, 2.1284682750701904, 0.40657860040664673, 1.129550576210022, -0.6562879085540771, 1.3868281841278076, -0.5671058893203735, 0.10433199256658554, -1.479398488998413, 0.3630703091621399, 0.731853723526001, -0.12405253201723099, 0.2770279347896576, 0.6598244905471802, 0.4747857451438904, -1.1947892904281616, -1.3667960166931152, -1.076537847518921, 1.5180399417877197, 0.23426954448223114, 0.05922738090157509, 0.1834670901298523, -0.9432873129844666, -0.08740843087434769, -0.8507233262062073, -0.20668593049049377, -0.3334406316280365, -0.9618349075317383, 0.007496733218431473, 0.4781697690486908, 0.8171724081039429, -0.0926203653216362], [0.7064865231513977, 1.1764756441116333, 0.3537314236164093, 0.2945033609867096, 1.3205004930496216, -1.9507100582122803, 1.6884530782699585, -1.734784483909607, -1.0919626951217651, -0.9746865630149841, -1.0086438655853271, -0.2280411422252655, -0.4339368939399719, 0.9002059698104858, -1.3724842071533203, -1.1974183320999146, -2.391010284423828, -0.1164950504899025, -0.5215909481048584, 0.03781208395957947, -0.15763629972934723, -0.10481777042150497, 0.952241837978363, -0.6941289901733398, 0.7857941389083862, 0.2889988422393799, 0.17399469017982483, 0.6260025501251221, 0.30950987339019775, -0.900405764579773, 0.13683630526065826, -0.09284657984972, 0.33514031767845154, -0.3249637484550476, -0.31949758529663086, -1.782650113105774, 0.2392614185810089, -0.7064831852912903, 0.42929011583328247, -0.31160423159599304, -0.7947705984115601, -0.06938625872135162, -0.5054537057876587, -0.7290782928466797, 0.3558950424194336, 0.7365221381187439, 0.7838229537010193, 0.3749607503414154, 1.4320425987243652, 0.13481035828590393, -0.2671639621257782, 0.21590054035186768, 0.4686652421951294, -0.7028423547744751, 2.4716596603393555, 0.10683033615350723, -0.3539391756057739, 0.13636063039302826, -1.234518051147461, 1.0394090414047241, -0.12627731263637543, -0.37487301230430603, 1.0532634258270264, -1.218429446220398, 0.08735440671443939, -0.846866250038147, 1.1122792959213257, 0.6071751713752747, -0.13806715607643127, 1.350571632385254, -0.02058032900094986, 0.7959348559379578, -0.6865409016609192, -2.0945351123809814, 1.4157944917678833, -1.316514253616333, -0.3437105119228363, 0.5340897440910339, -0.9835451245307922, 0.84761643409729, 0.39226803183555603, -0.27058324217796326, 0.27223873138427734, 0.5495288968086243, -0.5580521821975708, 0.8390668034553528, 1.0683218240737915, -2.1265902519226074, 0.7022692561149597, 1.0034253597259521, -0.7301374673843384, 2.0924723148345947, -0.43340492248535156, 0.017647240310907364, -1.3277709484100342, -0.7931362986564636, 1.5401585102081299, 0.3363020122051239, -2.2612457275390625, -0.01235573086887598, -1.2372041940689087, 0.30495765805244446, 1.2220830917358398, -0.19160102307796478, 0.9895550012588501, -0.4660274088382721, 1.7503745555877686, -0.7617714405059814, -1.27378249168396, -0.7248087525367737, -0.21766091883182526, 0.23430071771144867, -0.316098690032959, -1.7536553144454956, -1.296625018119812, 1.0485634803771973, -1.1091418266296387, 0.34086090326309204, -1.799277901649475, -0.7386525273323059, -0.5467721223831177, 1.5807496309280396, -0.738172709941864, -0.4309655725955963, -0.9183440804481506, -0.968413233757019, 0.3042466342449188, 0.23188750445842743], [0.6629714965820312, -1.5424761772155762, -0.17024490237236023, 2.0592143535614014, -0.5771197080612183, -1.1660364866256714, -0.4682371914386749, 0.8070078492164612, -1.0154246091842651, -0.6199190020561218, -0.735388457775116, -0.932923436164856, -0.6107611060142517, 0.2541278004646301, 0.7637208104133606, 0.8823776841163635, 0.15190647542476654, 0.9337490200996399, 0.3895408511161804, 0.8703030943870544, 0.5602406859397888, -0.3701317608356476, 0.24840770661830902, 2.0958776473999023, 0.22391092777252197, -1.3780310153961182, -0.5044471621513367, -0.47759711742401123, -0.45187652111053467, 0.10066177695989609, -1.2192951440811157, -1.0200223922729492, -0.25009194016456604, 0.12294970452785492, -0.22526875138282776, -0.2695176303386688, 1.2231812477111816, -1.884125828742981, 1.0999096632003784, -0.8983811736106873, -0.23992718756198883, -1.163885474205017, -1.224617600440979, 0.5912274718284607, -1.4647365808486938, -0.13315241038799286, 0.07454496622085571, -0.32034510374069214, -0.8631030917167664, 1.597350835800171, -1.3768703937530518, 0.2708432674407959, 1.4748961925506592, -0.03873905539512634, 0.3451329171657562, 0.03334713354706764, 0.37670478224754333, -0.16873084008693695, 1.0622965097427368, 0.9372450113296509, -0.18829137086868286, -1.2460185289382935, -0.42620187997817993, -0.4964478611946106, 1.2737340927124023, -0.3712119460105896, 0.38559582829475403, -1.6136434078216553, -2.1248724460601807, -0.5554711818695068, 1.0160601139068604, -0.950685441493988, -1.149733066558838, -1.1986230611801147, -0.43617236614227295, -1.5596888065338135, -1.1333281993865967, 1.296168565750122, 1.1757718324661255, -1.510191559791565, -0.6517139077186584, 1.9851200580596924, 1.4348971843719482, -0.5467853546142578, -1.903809666633606, -0.07855584472417831, 0.6156954169273376, -0.37528708577156067, -0.9413949251174927, 0.08579231053590775, 0.5739161372184753, -0.7159076929092407, -1.831561803817749, 0.9302056431770325, -1.0694447755813599, 0.21641606092453003, 1.6313787698745728, -0.7498616576194763, 0.23025134205818176, 0.6203989386558533, 1.6944953203201294, 0.7774879932403564, 1.4466913938522339, 0.7881951928138733, -0.7668471932411194, 1.723503589630127, -0.9258549809455872, -0.508813738822937, -0.78703773021698, 0.8967825174331665, -1.1632713079452515, 1.9421840906143188, -1.1151401996612549, 0.07931330800056458, -1.2352073192596436, 0.37089109420776367, 1.3822367191314697, -1.3101418018341064, -0.43100374937057495, 0.7579107284545898, 0.8204591274261475, -0.516977071762085, 2.3267264366149902, 2.7863290309906006, 0.9865851402282715, -1.5139390230178833, -1.0529228448867798, -1.5790307521820068], [-0.11951342225074768, -0.8754047751426697, -1.4830164909362793, 1.2360033988952637, -0.15832526981830597, -0.21300087869167328, -1.071480631828308, 0.09801142662763596, -2.0349414348602295, 0.09192343056201935, 0.3036113381385803, -0.17364582419395447, -1.2824491262435913, -0.7968083024024963, 1.3594305515289307, -0.4044622480869293, -2.027223825454712, -1.571723222732544, -1.6780426502227783, 0.6395906209945679, 0.15092021226882935, -0.9443530440330505, 0.6518012285232544, 0.5479869246482849, 0.03807566314935684, 0.5235965847969055, 0.4209192991256714, -0.6162900328636169, -1.893445372581482, -1.8561335802078247, 1.131379246711731, -0.03033686801791191, 0.3484955132007599, 0.7247894406318665, -0.8099991083145142, -0.7220434546470642, -0.780087411403656, -1.0990194082260132, -0.30610233545303345, -0.21582089364528656, -1.0050076246261597, -0.4452783465385437, -0.20498931407928467, 0.35028883814811707, -0.7916945219039917, 1.2447919845581055, -1.2983119487762451, -0.7362827658653259, 0.1160702258348465, -0.09538530558347702, -0.6209594011306763, 0.7601005434989929, -0.5436416864395142, -0.5037354826927185, 0.5740318894386292, 1.3170850276947021, 0.5798243284225464, 0.07987494021654129, 2.160609006881714, 1.0271937847137451, -0.7917320728302002, -0.4403468668460846, 0.36302322149276733, -0.2712448835372925, -0.1565679907798767, -0.9252164363861084, 0.2351434975862503, -0.022375892847776413, 0.4592900574207306, -0.8292474150657654, -1.1304765939712524, 0.4857981503009796, -2.0295827388763428, -1.7803024053573608, -2.7162857055664062, -0.6069526672363281, -0.0458226352930069, 0.29511651396751404, 0.023483678698539734, -0.6567687392234802, 1.3615261316299438, 0.5242623090744019, -0.2597314715385437, -2.1647582054138184, 0.09626226872205734, 0.36305466294288635, 1.4626009464263916, -1.967166781425476, -0.9150981903076172, 2.055642604827881, 0.5384026765823364, 0.31208470463752747, 0.2991649806499481, 0.6484252214431763, 0.31390732526779175, 0.6542214155197144, 0.5608898401260376, 1.380125641822815, 0.32202380895614624, 1.1579868793487549, 0.16552211344242096, 0.2801377475261688, -0.7870245575904846, -0.7618148922920227, 0.8981662392616272, 0.508488655090332, -0.7833918333053589, -0.8912780284881592, 0.24862739443778992, 1.0047610998153687, -0.043935392051935196, -0.7179407477378845, -0.07507377862930298, 1.3480629920959473, 0.021895363926887512, 1.6636019945144653, 0.057208720594644547, 0.23768959939479828, -3.0295116901397705, -0.05082975700497627, -0.3462686240673065, 0.1823350042104721, 2.962412118911743, -0.1778475046157837, 1.3807145357131958, 0.3658243715763092, -0.45783916115760803, 0.11679752916097641], [0.7511091232299805, 0.6562480330467224, 0.23697905242443085, 0.054506704211235046, -0.19563919305801392, -1.1626365184783936, -2.333488702774048, -1.5509557723999023, 0.1547730267047882, -1.3555026054382324, 0.7653663754463196, -1.24690842628479, -0.23847299814224243, 0.06156817078590393, -0.5049651265144348, -0.4857489764690399, -0.7598157525062561, 0.3089573085308075, 0.5404300093650818, -0.36695346236228943, -0.3575601577758789, -0.6350598931312561, 0.4330834746360779, 0.49065297842025757, -0.29058119654655457, 0.19403746724128723, -0.645831286907196, -1.3666006326675415, -0.43571460247039795, 0.3359711170196533, -0.9633684158325195, 0.929318904876709, 0.6756474375724792, 0.9221687912940979, -1.6341228485107422, 0.5978209376335144, 0.14616303145885468, 1.016175627708435, -0.20702669024467468, -1.1790692806243896, -0.5980297327041626, 2.246912956237793, -1.6285452842712402, 0.8344954252243042, -1.3393566608428955, -1.032881498336792, -0.14726144075393677, 1.4854646921157837, 1.1522501707077026, -1.5658764839172363, -0.2595078647136688, -0.7658659815788269, -0.07234556972980499, -1.7249399423599243, -1.6248759031295776, -0.5035088062286377, -0.26735374331474304, 0.7259225249290466, -0.29520148038864136, 0.4789138436317444, 0.33459293842315674, -2.304093360900879, 0.03632044419646263, 0.6219327449798584, 1.5409218072891235, 2.1337339878082275, -0.4886571168899536, 0.4195937216281891, 0.7974591851234436, -0.3915258049964905, -1.0457485914230347, -1.0758159160614014, 0.7327509522438049, -0.3538793623447418, 2.176435708999634, -0.07312063872814178, -0.5344631671905518, 0.7901189923286438, 0.492138147354126, -1.1769300699234009, 0.7666853666305542, -0.01953931897878647, 0.850861132144928, 0.23458901047706604, -0.31055891513824463, -0.3125625550746918, -0.2695866823196411, 0.6480415463447571, -0.6031395196914673, -0.5270475149154663, 0.12856678664684296, 0.10873623937368393, -0.9360108971595764, 1.325186014175415, 0.6785508990287781, -0.090887151658535, 0.19448311626911163, -0.24381764233112335, -1.0404468774795532, 0.1658141165971756, 1.9138057231903076, -0.4932156205177307, -0.6204642057418823, -1.0881277322769165, -0.0254130307585001, -0.6384538412094116, -1.7381329536437988, -0.01651846431195736, 0.43791165947914124, -1.8951612710952759, 1.1716725826263428, 0.6854058504104614, 1.2860674858093262, -0.8535226583480835, -0.03137429431080818, -1.293586015701294, -2.3908679485321045, -0.8682246804237366, 0.9163094162940979, -0.1676916480064392, -0.15833376348018646, -0.16849367320537567, 0.026545917615294456, -2.9425666332244873, 0.5738319158554077, 0.5928924679756165, 0.1470668762922287, 1.8801909685134888], [1.566630244255066, -1.297906517982483, 1.4577873945236206, -1.4316293001174927, 0.5374217629432678, 0.14930656552314758, -0.9806929230690002, 0.15055806934833527, 0.08896303921937943, -0.034890323877334595, -0.4617496430873871, -0.42945677042007446, -1.1183359622955322, -1.4646904468536377, -0.9776597619056702, 0.685247004032135, 1.5979912281036377, -0.01632213406264782, -0.19246378540992737, 1.6505597829818726, 0.1339527666568756, 0.5087787508964539, 0.547678530216217, -0.03618606925010681, -1.5267257690429688, 0.6819407343864441, -0.2237502932548523, 0.6866661310195923, -1.1461818218231201, -0.40062981843948364, -0.5004865527153015, -1.2629563808441162, 1.2989870309829712, -0.6383994817733765, -0.07566779106855392, -0.13680757582187653, -0.10383359342813492, -1.6839834451675415, 1.55570650100708, -0.6361481547355652, 2.17596697807312, 0.6152699589729309, -0.2009383887052536, 0.19388842582702637, 0.5828717947006226, -0.29556921124458313, -0.3997981548309326, -0.3251916468143463, 1.2966177463531494, 0.009717727079987526, -0.7098320126533508, -0.15630804002285004, -0.15988405048847198, -0.6457483172416687, 0.0735374391078949, -0.1540084034204483, 0.6800790429115295, -0.11363749951124191, -0.13192927837371826, -0.22964206337928772, -0.4843633770942688, -1.1384470462799072, -0.9326533079147339, 0.8371725678443909, 1.023270606994629, 0.8257899880409241, -0.8537322878837585, -0.6907775402069092, -0.7939041256904602, -2.1871161460876465, -1.0175701379776, 1.2337287664413452, 0.7068626880645752, -1.2514859437942505, 1.1116951704025269, -2.0904390811920166, -0.4526495337486267, 1.245459794998169, 2.1326401233673096, 1.3565635681152344, 0.3689878284931183, 0.2739306390285492, 0.7120552062988281, -2.2147085666656494, 0.5785764455795288, 1.5318363904953003, 0.42534127831459045, -1.80707848072052, 1.9844629764556885, -0.651370108127594, -0.13704587519168854, -0.9695618748664856, 0.8575971126556396, -0.5580468773841858, -0.7933425903320312, -0.3357572555541992, -1.3106303215026855, 0.11568030714988708, 0.7498899698257446, 0.5989921689033508, -1.7561204433441162, 1.8465358018875122, -0.4873165488243103, -0.3384323716163635, -0.816257894039154, 0.8022164106369019, 0.9386136531829834, 1.1867469549179077, 2.3914544582366943, -0.059088412672281265, -0.5758297443389893, 0.240123450756073, -0.5313227772712708, 0.5910547971725464, -0.29562610387802124, 0.2356543391942978, 0.6217742562294006, 0.7967365384101868, -1.305288553237915, -0.06930964440107346, 0.45226699113845825, 0.5641772150993347, -0.1791602373123169, -1.2556912899017334, 1.6848779916763306, -0.7854236364364624, 1.3339415788650513, -0.332387238740921], [-0.5472126603126526, -0.6983315944671631, -1.5392217636108398, 0.07124804705381393, -0.7286600470542908, -1.0762290954589844, 0.09096631407737732, 0.01915290765464306, -0.41958683729171753, 0.6202212572097778, 0.442914754152298, 1.3464107513427734, -1.0293171405792236, 0.31198132038116455, -0.7942719459533691, -0.06465616822242737, -1.5940877199172974, -0.946142315864563, -0.10591944307088852, -0.6582507491111755, -0.4795167148113251, -0.9031358361244202, 0.3077089190483093, 0.36855143308639526, 0.25035178661346436, 1.111741065979004, 0.49708032608032227, -0.2864929437637329, -0.6881508827209473, 0.009881916455924511, 0.5077013373374939, -0.9015211462974548, -0.5019708871841431, 0.6324177980422974, 0.3656822144985199, -0.8932254314422607, -1.9109605550765991, 1.4296237230300903, 0.09394074231386185, -0.0075492849573493, 0.018761973828077316, 0.6109524965286255, -0.2122918665409088, 0.5243521928787231, -0.1100437194108963, 0.5296132564544678, -0.19581249356269836, 0.14321789145469666, -0.07898853719234467, 0.30893296003341675, 0.31700652837753296, 0.9608911871910095, 2.3718247413635254, 0.457732617855072, -0.38034787774086, 1.0663632154464722, 0.1487794667482376, -1.489660620689392, 0.9713208675384521, -0.05396192893385887, 1.0404672622680664, 0.02400762401521206, -0.05835878849029541, -0.06054497882723808, 1.2963074445724487, -0.5679718255996704, 1.1850022077560425, -0.05263308435678482, -0.718139111995697, -0.0249774307012558, -0.16590824723243713, -1.7132595777511597, 1.5905495882034302, 0.36181363463401794, 1.1110866069793701, 1.8089916706085205, -0.3279261589050293, -0.6856260299682617, 0.4429871141910553, -0.080536849796772, -0.5614732503890991, -1.0067919492721558, -1.0489505529403687, 0.7566754221916199, -0.6550727486610413, -0.4145353138446808, 0.7912958264350891, -0.2179972380399704, -0.36003753542900085, -0.039745528250932693, 0.32183393836021423, -0.22378505766391754, 1.4686518907546997, 1.305302619934082, -0.3947727382183075, -1.6706573963165283, 1.0079575777053833, -1.2871463298797607, -0.17269086837768555, 1.9544751644134521, 0.41884511709213257, -0.6173960566520691, 1.7438346147537231, -0.2875271737575531, 0.46043166518211365, -0.3414105176925659, -0.6930782794952393, -0.6154007911682129, -0.6648003458976746, 0.8715904951095581, 1.115127444267273, -0.9762346744537354, -0.1760515719652176, 0.5389069318771362, 0.23154792189598083, -0.7121157050132751, -1.8045804500579834, -0.9497490525245667, 1.0254497528076172, 0.8912012577056885, 0.40713584423065186, 1.3786249160766602, -0.9270107746124268, -1.4719634056091309, -0.6882910132408142, 0.7473226189613342, 1.3379905223846436, 0.4362519383430481], [1.288375735282898, -0.98176509141922, -0.8312051296234131, 0.8252087235450745, 2.9021804332733154, -0.19827522337436676, 0.8335146903991699, -1.6302095651626587, -1.3730605840682983, -0.1513383984565735, -0.9200433492660522, 0.0038529899902641773, -0.25057610869407654, 1.1041163206100464, -0.017602985724806786, -0.8903554081916809, -0.43234747648239136, 0.40324363112449646, -0.3820398449897766, -0.6278716325759888, 1.0179364681243896, -1.5410070419311523, -0.36595940589904785, 0.5769956707954407, -0.30605071783065796, -0.45820048451423645, -0.15738408267498016, 0.017395516857504845, 0.4730028510093689, 0.6116286516189575, 0.47118479013442993, -0.35624587535858154, -0.09005342423915863, 0.3723534941673279, 0.4268541932106018, -1.3919926881790161, 2.5031518936157227, 1.6419960260391235, 1.5023720264434814, -0.4391598105430603, 0.06386484950780869, -0.24233375489711761, -1.3240329027175903, -0.06525196880102158, 1.1073036193847656, 0.5569736957550049, -0.05247916653752327, 0.7459070086479187, 0.8776192665100098, 0.9171374440193176, -0.41932135820388794, -2.295727014541626, -0.536362886428833, -1.4576377868652344, -0.30804532766342163, -0.37527263164520264, -0.347398579120636, 0.8925096392631531, -0.5074843168258667, -0.7572633624076843, 0.05656052753329277, 1.8482842445373535, -0.04212187975645065, -0.4746818542480469, -1.822091817855835, -1.5282291173934937, -1.148132085800171, 0.20410512387752533, -1.2519577741622925, -0.13501615822315216, 1.1585880517959595, 0.9853958487510681, -0.6221580505371094, 0.5115012526512146, -1.8056349754333496, -0.11742597818374634, 0.38766688108444214, 0.24627049267292023, -2.355476140975952, -0.9909470677375793, -0.05997297540307045, 0.7007870078086853, 0.9149301648139954, -2.2815284729003906, 1.301574468612671, -1.773964524269104, -0.7056340575218201, -1.289157748222351, 1.8615317344665527, -1.0983365774154663, 1.4421498775482178, -0.48994454741477966, -1.0975375175476074, 2.238579750061035, 1.0626616477966309, 1.8122223615646362, 0.6466630697250366, 0.8923087120056152, -0.39698880910873413, -0.33125028014183044, -0.18924309313297272, -0.019755655899643898, 0.06224367022514343, -1.8171579837799072, -0.7752475142478943, 1.845279335975647, 0.11164841055870056, -0.34672901034355164, 0.7461231350898743, -0.5858049392700195, -0.8848742246627808, 0.8109684586524963, -1.0458894968032837, 1.1718635559082031, 0.2718575894832611, -1.9952812194824219, -2.101306200027466, 0.5708433389663696, 0.10002779215574265, 0.18897178769111633, -1.4034881591796875, 2.233604907989502, -1.7037110328674316, 0.4858889579772949, -0.4695599377155304, -0.5773443579673767, -0.6130088567733765, 0.007476722355931997], [-0.4239672124385834, 0.05693100020289421, 0.09961040318012238, -1.914297103881836, -0.09958526492118835, -0.5041038990020752, 1.4599874019622803, 0.5168346166610718, -0.33774590492248535, -1.1146001815795898, 0.8788749575614929, 1.4922597408294678, 0.2992302477359772, 0.06160929426550865, 0.9428545236587524, 1.754511833190918, 0.8608289957046509, 0.13486279547214508, 1.7576279640197754, -0.497898131608963, -0.340817391872406, 1.168424367904663, 0.1609923392534256, 0.3321841061115265, 0.17484989762306213, -1.1104156970977783, -0.8376626372337341, -0.12076674401760101, 0.8878412842750549, 1.2720633745193481, -0.39656776189804077, 0.34458673000335693, 1.3749797344207764, 0.7460612058639526, -1.072158694267273, 0.6570891737937927, 1.4402505159378052, 1.4305123090744019, 0.31927305459976196, -1.1814497709274292, -0.6234802007675171, 2.1592354774475098, 0.30797433853149414, -0.37787574529647827, 0.5674578547477722, -0.2032611221075058, -0.18755996227264404, -0.644198477268219, 0.5148499608039856, -1.080600380897522, 0.4023488163948059, -0.45400020480155945, 0.593913197517395, 0.04519464075565338, -0.16635163128376007, -1.2827621698379517, -0.01983199454843998, 0.17666254937648773, -1.346010446548462, -1.5905394554138184, 1.3586992025375366, -1.4066346883773804, 0.5116982460021973, 0.9842186570167542, 0.46687766909599304, -0.30103570222854614, -1.3521438837051392, -0.7929726243019104, -0.8190770149230957, 0.4098740816116333, 0.5472109317779541, 0.6356373429298401, 0.14044500887393951, -0.11393719911575317, 1.919641375541687, -0.1499430388212204, -0.9552225470542908, -0.2553446888923645, -1.3533375263214111, 1.7631111145019531, -0.2534047067165375, -1.1861999034881592, 0.12142636626958847, -0.5152177810668945, -0.2557893693447113, 1.2780448198318481, 1.4875527620315552, 1.7754846811294556, 0.8339847922325134, 0.24331465363502502, -0.15829122066497803, 0.009962616488337517, 1.1655277013778687, -0.5864336490631104, -1.2073994874954224, 1.5079094171524048, 0.6274088621139526, -0.8788926005363464, -1.583044409751892, 1.6254606246948242, 0.16540054976940155, -0.20087043941020966, 0.3479781150817871, 0.6027886867523193, 0.4239651560783386, -0.7058084011077881, 1.4360661506652832, -0.9430203437805176, 2.18709659576416, -0.4660324454307556, 0.6175848841667175, 0.574390172958374, 0.81128990650177, -0.3641253411769867, 0.05986867845058441, -1.133568286895752, 0.18003523349761963, 0.24600036442279816, 0.9819772839546204, -0.05680602788925171, 1.0518324375152588, 0.2039523720741272, 0.1304449588060379, 0.9813315868377686, -0.3476976454257965, -0.9096102714538574, 0.47151824831962585, -0.12117175757884979], [-1.5170246362686157, 1.5933868885040283, 1.3922913074493408, 1.622408151626587, -1.8687467575073242, 1.6774795055389404, 0.015965012833476067, 1.4440499544143677, 0.4844783842563629, -1.0751749277114868, -0.06478102505207062, -0.6026446223258972, 1.607668399810791, -1.7964774370193481, 0.868257462978363, 1.7610960006713867, 1.2761616706848145, 0.1562115103006363, 0.11588720232248306, -0.03516291826963425, -0.5576388835906982, 1.0043070316314697, -0.28241977095603943, 1.5789872407913208, 0.8075563907623291, 1.2645199298858643, 0.2588770091533661, -0.004951391369104385, -1.5223612785339355, 0.16034162044525146, 1.0154591798782349, 0.806267261505127, 1.3925613164901733, -0.4507905840873718, 1.175468921661377, 0.634124219417572, -0.13403379917144775, 0.0038557627703994513, 0.05422446131706238, -0.6019442677497864, 0.11926736682653427, -0.21606586873531342, 1.111326813697815, 0.08893487602472305, -0.1549050360918045, 0.2606980502605438, -2.1066691875457764, -1.6460742950439453, -0.14068388938903809, 1.1354690790176392, 1.1346603631973267, 0.46809983253479004, -0.2198605090379715, -0.5602189898490906, 1.6632188558578491, -0.6571811437606812, -1.41387140750885, -0.09772565960884094, 0.11460709571838379, -0.6168971061706543, -1.198598027229309, 2.7905116081237793, -0.403387188911438, 0.013094200752675533, 0.13812632858753204, 1.113398790359497, -0.3660166561603546, -0.5591640472412109, -1.8624475002288818, -1.1944822072982788, 0.5500497221946716, 1.2367093563079834, 0.7649884223937988, 0.8518192172050476, -0.5800442099571228, 0.9264968633651733, 0.016636621206998825, 1.0090076923370361, -1.4854871034622192, 1.9644572734832764, -0.6439628005027771, 0.33708807826042175, -0.3135727047920227, -0.7527339458465576, -0.7724292278289795, 0.01631574146449566, -1.1310696601867676, 0.012130589224398136, 0.21990256011486053, 0.23357589542865753, 0.544758141040802, -0.12042059749364853, 1.0373337268829346, 0.8356124758720398, -0.48482587933540344, 0.2210012525320053, 0.7054213881492615, 1.802086591720581, 0.038794972002506256, -0.9769253730773926, 0.3243238031864166, -0.1950012445449829, -0.7571503520011902, -0.8021544814109802, 0.320659875869751, 0.2869836986064911, 1.3821262121200562, 1.412866473197937, 1.221376657485962, -0.9511991739273071, 0.08293728530406952, -1.1316356658935547, 1.142255425453186, -0.7306783199310303, -2.752847194671631, 0.17294128239154816, 1.0452982187271118, 0.9348305463790894, -0.5085421204566956, -0.2189805805683136, 0.20649118721485138, 0.3768576979637146, -1.7284798622131348, 0.5793799757957458, -0.0316297672688961, 0.8287093043327332, 0.18161652982234955, 0.6557677388191223], [0.37719520926475525, 0.7324533462524414, 0.09283362329006195, 0.012437738478183746, 0.8611526489257812, 0.9461947679519653, 0.011573837138712406, 0.6146681308746338, 1.3545122146606445, -2.296734094619751, 0.9467191696166992, -1.7050427198410034, -0.9233577251434326, -0.0719752237200737, 0.20690636336803436, 0.19813624024391174, 1.645136833190918, -0.17245109379291534, -0.2173466682434082, 0.08138500154018402, 1.1392467021942139, 0.21868100762367249, -1.7656983137130737, 1.7817769050598145, 1.8854492902755737, -1.2547374963760376, -1.4050862789154053, -0.1138293594121933, -1.7922587394714355, -1.132075309753418, -3.3285508155822754, 0.06853537261486053, 1.3715473413467407, -0.8515796065330505, -0.27307069301605225, 0.4143112301826477, 0.45149898529052734, -0.7885679602622986, 0.20673830807209015, 1.1869678497314453, -2.1696794033050537, -0.7725386619567871, 0.8569465279579163, 0.05011787265539169, -0.04358132928609848, 0.02665127068758011, -3.0629055500030518, -0.3004228174686432, 0.11874707788228989, 1.5007562637329102, -1.4372141361236572, 0.20348036289215088, 2.0989174842834473, 1.0684312582015991, -0.3868369162082672, 2.6112303733825684, 0.4860480725765228, 0.6467059254646301, -0.03995503857731819, -0.8792308568954468, -0.6363914012908936, 0.527807891368866, 0.5360599756240845, 0.8843895196914673, -0.15644071996212006, -0.6048191785812378, -1.1687675714492798, -0.2953270673751831, -0.634975254535675, 2.7613327503204346, 0.6016597747802734, -0.0724434182047844, 0.6705947518348694, 0.2858634889125824, -2.1366708278656006, 0.36136049032211304, 0.9271226525306702, 0.3645363450050354, -0.3870491087436676, 0.6782261729240417, 0.33217912912368774, 1.1388951539993286, 0.41116026043891907, -0.9995376467704773, 0.6980440616607666, 0.7924396395683289, 1.3371912240982056, 0.6930931806564331, 0.6497889161109924, 0.9397222995758057, 0.8545110821723938, -1.3735618591308594, -0.35516202449798584, -0.7303785085678101, -1.2685232162475586, 0.3350428342819214, 1.710991382598877, -0.8577632904052734, -1.1097843647003174, 0.6531751155853271, -0.6436483860015869, -0.2638367712497711, 0.057098064571619034, 0.4072476029396057, 1.6979951858520508, -0.17600464820861816, -1.1850382089614868, -1.1550276279449463, -1.4562908411026, -0.8779016733169556, -0.3409384787082672, -0.23919521272182465, -0.7521978616714478, -0.34549012780189514, -0.40113604068756104, -1.1855053901672363, -1.30890953540802, -0.05108442157506943, -1.0713471174240112, 1.2315655946731567, -0.705906867980957, -0.9721895456314087, -0.9624289870262146, 1.0652351379394531, -1.9418519735336304, -0.014719987288117409, -1.4834321737289429, 2.3677749633789062], [0.19408981502056122, -1.506183385848999, -0.7524001002311707, -1.2153337001800537, -1.4417203664779663, 0.013079913333058357, 1.2930972576141357, -0.348271906375885, -1.396031379699707, -0.789891242980957, -0.5899997353553772, 0.8137831687927246, 0.23837685585021973, 0.6693428158760071, 1.0566240549087524, -1.9293713569641113, -0.457582026720047, 1.1203030347824097, 0.6325560808181763, 1.0116499662399292, 0.9057405591011047, 0.34362688660621643, 0.14915746450424194, 0.9043241739273071, 1.5186448097229004, 0.7286170721054077, -0.5064175724983215, -0.39943376183509827, -0.015394637361168861, 1.6437424421310425, -0.41063469648361206, 0.007782906759530306, -0.29062724113464355, 0.1872342824935913, -0.8912110328674316, -0.45696136355400085, -1.0630302429199219, -0.3386214077472687, -0.7818748354911804, -1.4085503816604614, 1.5802854299545288, -0.2599746584892273, 0.8639221787452698, -0.6157633066177368, -0.2514253556728363, 0.18745173513889313, -0.6303765773773193, -1.424839735031128, -0.8531500697135925, 1.0682373046875, 0.5549343824386597, 0.40391984581947327, -0.9284195899963379, -0.7103203535079956, -0.5189930200576782, -0.5883902907371521, -0.5343596935272217, 0.519457221031189, -0.14277255535125732, 0.4708058536052704, -0.39824098348617554, 1.6084206104278564, 0.8757684826850891, 1.902445673942566, -0.36388418078422546, -0.15135306119918823, 2.267200469970703, -0.2662573754787445, -0.06669288873672485, 1.0392091274261475, -1.6721854209899902, 0.33403709530830383, -0.07063012570142746, 1.6726428270339966, -0.24122236669063568, -0.07705815136432648, -0.7546887993812561, 0.8377367854118347, -0.5671581029891968, 0.8624200820922852, -1.6204203367233276, -0.4650459885597229, 0.0935298278927803, -1.5833901166915894, 0.002761654555797577, -0.848313570022583, 0.01608883962035179, 1.506123423576355, 1.5123975276947021, 0.5406444668769836, -2.175300121307373, 1.0533642768859863, 0.9904182553291321, -0.6714382767677307, -0.4977816641330719, 2.338176965713501, 0.3063614070415497, -0.8697240352630615, -0.8663873672485352, -0.5859913229942322, -0.6349615454673767, 2.264780282974243, 0.7001528143882751, -0.11077529937028885, -1.4906418323516846, 0.2378121316432953, 0.7105748653411865, -0.9511036276817322, -0.20116931200027466, 0.6675289869308472, -0.24218952655792236, 0.5629411339759827, -0.005777809303253889, -1.0963778495788574, 1.0857230424880981, 1.0231070518493652, -0.6284170150756836, 1.3969743251800537, 0.8517525792121887, -1.3232494592666626, 0.48152074217796326, -0.6314257979393005, 0.40021347999572754, 0.6642886996269226, -0.5358449220657349, 0.13884799182415009, 0.2012867033481598, -0.5905615091323853], [2.006892681121826, 0.9750805497169495, -1.3615859746932983, 2.846271276473999, 0.9591102600097656, -0.539800226688385, -0.7185769081115723, 1.588210105895996, -0.6108387112617493, 0.20238183438777924, -0.49479940533638, -0.47685983777046204, 0.6878519058227539, 0.8233654499053955, -0.5294613242149353, 0.18724371492862701, -0.058101411908864975, -0.46583291888237, 1.7593977451324463, -0.40892380475997925, 1.8589597940444946, -0.3181673586368561, 0.11497765779495239, 1.6551315784454346, 1.8911542892456055, 1.1134319305419922, 0.11588779091835022, 2.5850887298583984, 1.348889946937561, 0.8309285640716553, -0.18354389071464539, -1.3026134967803955, -0.8411389589309692, 1.339989423751831, 1.3691030740737915, 0.5411278605461121, -0.10984520614147186, 0.43869543075561523, 0.4150584638118744, -0.3240405321121216, 0.3114457428455353, 0.25947779417037964, 0.4759576618671417, -0.9130337238311768, 1.3805040121078491, -0.978202223777771, 0.563878059387207, 0.10148786753416061, 0.9223639965057373, 0.11002963036298752, -2.849001407623291, -0.8741633892059326, -1.6144239902496338, -0.6314433813095093, 0.5936667919158936, -1.2981630563735962, 0.3135778307914734, 0.48131924867630005, -0.4239670932292938, 1.3333908319473267, -0.2549521327018738, -0.46498721837997437, 0.7827878594398499, 0.9289963245391846, -0.8786349892616272, -0.5682740211486816, 0.44888368248939514, -0.32195621728897095, -0.6230401992797852, 0.3949590027332306, -0.2532801330089569, 0.012874532490968704, 1.334200143814087, -0.45201852917671204, -0.9598793387413025, -0.9478906989097595, 0.872622013092041, 1.8393347263336182, 1.3454631567001343, -1.060436725616455, 0.29347100853919983, -1.596848726272583, 0.02023405022919178, -1.5442527532577515, -1.5291922092437744, 2.7701783180236816, -1.0503126382827759, 0.0543857105076313, 0.4630657434463501, 0.5570018291473389, 0.3006303012371063, 1.737899899482727, -1.0009890794754028, -0.6508757472038269, 0.8370084166526794, 1.5006866455078125, 0.02508893609046936, 1.4821277856826782, 0.7261348366737366, -0.23187677562236786, -0.1718614399433136, -0.11519438773393631, 0.7452957630157471, 0.7882619500160217, -0.6089359521865845, 0.8195750713348389, 0.18830731511116028, 0.817782998085022, -0.12337836623191833, 0.29466477036476135, 0.8741042613983154, 0.26612335443496704, 1.0022464990615845, 0.7685666680335999, 0.021880989894270897, -1.1293702125549316, -0.13424566388130188, -1.9544339179992676, 1.4230343103408813, 0.5332143306732178, -0.36783546209335327, -0.3546612858772278, 0.021345296874642372, 0.11462269723415375, -0.5934709310531616, -1.3945082426071167, -0.7818652391433716, -1.7916978597640991], [0.2107315957546234, 0.11823957413434982, -0.310944527387619, 0.8084680438041687, 0.1178646981716156, -0.3595133125782013, 0.05904146283864975, 0.15859465301036835, 0.4326343238353729, 0.535369873046875, 1.255162000656128, 0.26505741477012634, -0.874890148639679, -0.32835936546325684, -0.5862587094306946, -1.1385730504989624, -3.4209508895874023, -0.9513847231864929, 0.019286703318357468, -0.20262345671653748, -0.054419033229351044, 0.2638193666934967, -0.04395739361643791, 0.1261916607618332, 1.2989784479141235, 1.08579421043396, -1.1245733499526978, 1.4403892755508423, -0.5393693447113037, 0.35106074810028076, -0.2039412558078766, -0.9668379426002502, 0.30097511410713196, -1.8139772415161133, 0.21661320328712463, 0.5146923661231995, 1.4537070989608765, -2.5083718299865723, 0.0008648060611449182, -0.21695353090763092, -0.6419661641120911, -2.554001569747925, 0.976263701915741, -0.9491897225379944, -0.23792314529418945, -1.1370174884796143, 0.4795490801334381, -1.124985694885254, -1.2711957693099976, 1.2774338722229004, -0.1423889398574829, 1.1929563283920288, -0.31583699584007263, -2.7204864025115967, -0.4072484076023102, -1.542762279510498, 0.7186004519462585, 0.6451050639152527, -1.8211829662322998, -0.5566009283065796, 0.7624727487564087, 1.374787449836731, -0.3473857641220093, 0.2119995653629303, -0.9619017243385315, 0.7583380341529846, -0.5394787192344666, 2.055828332901001, 2.1864707469940186, -0.6278376579284668, 0.6787291169166565, -1.3019241094589233, -0.4261852502822876, -0.9947441220283508, -1.0897812843322754, -2.67010235786438, 0.025591326877474785, -1.3080883026123047, -0.8488807678222656, -0.16924680769443512, 1.531449317932129, -2.1493403911590576, -0.4112548232078552, -0.7838580012321472, -0.09370777010917664, -1.1043535470962524, 0.898918867111206, -0.398263156414032, 1.2747615575790405, -0.10165189951658249, -0.8059604167938232, 0.4304264485836029, 0.7819896936416626, -0.1744726300239563, 0.6389343738555908, -0.29378360509872437, -0.7994686961174011, 1.1704787015914917, -1.6524616479873657, -0.40337133407592773, -0.36460161209106445, 1.6454923152923584, 2.124285936355591, 0.28001126646995544, -1.3657894134521484, 0.6210984587669373, -0.585463285446167, 0.6910759210586548, -0.003373477840796113, 0.9361292123794556, -0.5380035638809204, 0.46555185317993164, 0.3696650564670563, 0.6145567893981934, 0.003562705358490348, 0.3321998417377472, -0.03246055915951729, 0.03466135635972023, 1.133362054824829, 1.1581897735595703, -0.11082196980714798, -1.963948369026184, -0.5930108428001404, 0.6826115846633911, -0.46437352895736694, 1.645494818687439, -1.4869142770767212, -0.87591552734375], [0.5956725478172302, -0.10492933541536331, -0.6204127669334412, 1.1939082145690918, 0.22074952721595764, -0.9023813605308533, 0.8486183881759644, -1.0955793857574463, -0.13423782587051392, 2.6157069206237793, 0.309032142162323, 0.9383787512779236, -0.7205201983451843, 0.6082671880722046, -0.845780074596405, 0.30129921436309814, -0.7638044357299805, -0.288563996553421, 0.11377642303705215, -0.7997925877571106, 1.4907147884368896, -0.09075289964675903, -2.043531656265259, -0.7165296077728271, -1.0116183757781982, -0.9114264249801636, -0.8704735040664673, 0.21687176823616028, -0.6354225277900696, -0.8827699422836304, 0.6454464197158813, 0.5844572186470032, -0.2774602174758911, -0.11461443454027176, 0.33367297053337097, -0.0909825786948204, -0.11775381863117218, 0.21929635107517242, 0.6466006636619568, -0.2744455337524414, -0.9078157544136047, 1.9549405574798584, 0.24699999392032623, -0.06293239444494247, 0.2331957370042801, 1.1998324394226074, 1.1824485063552856, 1.3179162740707397, -1.0820739269256592, -1.1525030136108398, 0.6526553630828857, 0.19130155444145203, -1.2184756994247437, 0.517909586429596, -0.5528398752212524, -0.11542891710996628, 0.7690523862838745, 1.6519465446472168, 1.0487525463104248, -0.5216500163078308, -1.322278380393982, 0.5097064971923828, 1.0971338748931885, 0.5463345646858215, 0.5690886974334717, -1.8890385627746582, 0.5954329371452332, 1.6572985649108887, -0.2840399444103241, -1.2998489141464233, 0.6908019185066223, -0.033111512660980225, -0.14434124529361725, 1.9181876182556152, -0.9832029938697815, -0.255542129278183, 0.5818867087364197, 0.038055140525102615, -0.007019537501037121, 0.39405959844589233, 1.3240946531295776, -1.7125194072723389, 0.04298793524503708, -2.6197426319122314, -0.6558846235275269, -0.04099370911717415, -1.564764142036438, -1.3311059474945068, 0.7998847365379333, -1.1509575843811035, 1.6096200942993164, -0.13779211044311523, -1.2976311445236206, 0.815812349319458, -0.4688519835472107, -1.447922706604004, -0.6898661255836487, 2.5467581748962402, 0.34764954447746277, 0.3442043662071228, -1.0644906759262085, 0.44122400879859924, 0.2250705361366272, -0.019215695559978485, -1.255181074142456, -0.9052125215530396, 0.7593598365783691, -0.21674995124340057, 1.108607292175293, 0.5319430828094482, -1.2966833114624023, -0.7306949496269226, 1.969781756401062, 0.31855618953704834, 0.5399361252784729, -0.31491923332214355, -0.7415160536766052, -1.2299036979675293, -0.18161359429359436, -0.513266921043396, -0.7637792229652405, -0.42627987265586853, -2.094487190246582, -0.3881342113018036, -0.28845661878585815, -0.6590648889541626, -0.7901232838630676, 0.19604314863681793]]], [[[-1.2589480876922607, -0.333595335483551, 0.0433078408241272, 0.5029807686805725, -3.092958688735962, 1.000345230102539, 1.2323379516601562, 0.5455424189567566, -0.3925662338733673, 0.029833631590008736, -0.8043987154960632, -0.5275653600692749, -1.0311115980148315, 1.0652141571044922, 0.6625593900680542, -1.0099374055862427, -0.2117978036403656, -0.7649903893470764, 1.3252818584442139, -0.10240162909030914, 0.04673723503947258, -1.057490348815918, -0.5560937523841858, -0.6835765838623047, 0.43160057067871094, -0.5985526442527771, -1.8270808458328247, 0.5215156078338623, -0.5338792204856873, 0.598850667476654, -0.9517288208007812, -1.1817152500152588, 1.8310071229934692, -1.7781412601470947, 0.15180452167987823, 0.8040915727615356, -0.8260695934295654, -0.6139710545539856, 1.3050848245620728, 0.8171901106834412, -0.04802175238728523, 0.3145899772644043, 0.07883918285369873, 0.18384724855422974, -1.300840973854065, 0.2931510806083679, -0.6961805820465088, -0.2812234163284302, -0.9203717112541199, 0.008755396120250225, -0.05235685408115387, -0.4899666905403137, -1.3914539813995361, -0.7782489061355591, 0.16609540581703186, -1.5272743701934814, -0.9870241284370422, -2.6156766414642334, 1.3817079067230225, 0.5672726035118103, 1.972762942314148, -0.8805143237113953, 0.7806776762008667, 0.8503566980361938, 0.3607763350009918, -1.2565973997116089, 0.8265348076820374, -0.7786139845848083, 0.47379475831985474, 0.5896202921867371, 0.41815540194511414, -0.48280763626098633, 0.6309332251548767, 0.015382209792733192, -0.31295549869537354, 0.4178391098976135, -0.8357042670249939, 0.43869805335998535, -0.15136316418647766, -2.0807371139526367, -0.3950473368167877, -1.1578925848007202, -0.09514809399843216, -0.588850200176239, -1.083373785018921, 0.5196468234062195, 0.8343771696090698, 1.0807819366455078, -1.007912516593933, -0.8052242398262024, -1.5238901376724243, -0.8246645927429199, 0.40916940569877625, 1.0822463035583496, 0.7634907364845276, -0.17089369893074036, -1.4417119026184082, 1.9985568523406982, -0.8297442197799683, -1.2567955255508423, 1.0116972923278809, -0.18524782359600067, 0.8117108345031738, -1.3263410329818726, -0.7339604496955872, 0.052247192710638046, 1.568963885307312, -1.180983304977417, -1.688767433166504, 0.9137154221534729, 0.5406140685081482, 0.8797073364257812, -0.10668456554412842, -0.19322305917739868, -1.552642822265625, -0.38780882954597473, 0.4041930139064789, -0.810660719871521, 0.7966102361679077, 0.35141029953956604, -0.3917590379714966, 0.48283350467681885, 0.07776182889938354, 0.5710077881813049, 0.7883771657943726, 0.4791220724582672, -0.6152142882347107, 0.2960992455482483], [-0.03910248354077339, -0.21234402060508728, 0.4604121148586273, 1.4811902046203613, 0.08596886694431305, -0.7194544076919556, 0.734809935092926, 0.3194722533226013, -0.13073016703128815, -0.5125994086265564, 0.6367749571800232, 1.2080628871917725, -0.74541836977005, 0.6589915156364441, 0.27318042516708374, -1.4916670322418213, 1.301628828048706, 0.8918095827102661, -1.1798121929168701, 0.43887150287628174, 0.3568715751171112, 0.9109583497047424, 1.7733298540115356, 0.5393761992454529, 0.7009121179580688, -0.09871482104063034, -1.3628066778182983, -0.14163292944431305, 0.3708605468273163, 0.2726007401943207, 0.7792450785636902, -0.5402709245681763, -1.0707710981369019, 0.8757749199867249, 1.0223366022109985, -0.3329051733016968, 0.10253246873617172, 0.010003340430557728, 0.3387434184551239, -1.4072093963623047, -0.6217184662818909, -1.6010370254516602, 0.5310805439949036, -0.6953514218330383, 0.2202187180519104, 0.22859366238117218, -0.9747300148010254, 0.22482112050056458, -0.2384144365787506, 0.589943528175354, -0.7602766752243042, 0.6424435973167419, 0.385090708732605, 0.2685520648956299, 0.7642807960510254, 0.9848360419273376, 0.3847109377384186, -0.19144511222839355, -0.46614503860473633, 1.8515334129333496, -0.39058926701545715, 0.18104547262191772, 0.3572697639465332, 0.4270821511745453, -0.543958842754364, 0.04572588950395584, -0.09837038069963455, 0.7370061874389648, 0.09517870098352432, 0.7801414728164673, -0.48486799001693726, -0.1642424613237381, 0.15354298055171967, 0.7432776093482971, 1.5844175815582275, 0.22291123867034912, -0.7920088171958923, 1.608475923538208, -1.1013333797454834, 2.803006649017334, -1.307973861694336, 0.6191278100013733, -1.1532888412475586, 0.5153228044509888, 0.4569258391857147, -0.12469421327114105, 1.1555067300796509, 0.8026435375213623, -0.3762166202068329, 1.2438926696777344, -0.07374967634677887, -0.5632079243659973, 0.5686811208724976, -0.2563685476779938, -0.4478386640548706, 1.2098371982574463, 0.5461686253547668, 0.5352182388305664, 0.09851513803005219, -1.2015305757522583, -0.20402677357196808, 0.4424302279949188, -1.1777563095092773, 0.13588018715381622, -1.7320054769515991, 1.4811698198318481, 2.417989492416382, 0.814992368221283, 0.4435266852378845, 0.11506394296884537, -0.35480374097824097, 1.2760887145996094, -0.35198473930358887, 1.1231136322021484, 0.3866141438484192, 0.9246235489845276, -0.900502622127533, -1.15192711353302, 0.6598271131515503, 0.09849180281162262, -1.4083460569381714, -0.9779816269874573, 0.12208129465579987, -0.9126855134963989, 1.0687061548233032, 0.17374713718891144, -0.403881698846817, 1.2228583097457886], [0.07465840131044388, -0.18400907516479492, 1.0548959970474243, 0.9491714835166931, 0.17803741991519928, 0.6280858516693115, -0.08240988105535507, -0.00022116197214927524, -0.48978734016418457, -1.3633369207382202, 0.1031574159860611, 1.5574359893798828, 0.5510792136192322, 0.2528291642665863, -1.3442423343658447, -0.5734999179840088, -1.2972992658615112, 0.2655472755432129, 0.1754433959722519, 1.332319974899292, 0.2873290479183197, 0.11084529012441635, -0.8161910176277161, -1.2104648351669312, -0.4854676425457001, -0.6953175663948059, -1.1269639730453491, 1.4437931776046753, -0.8397328853607178, -0.9740679860115051, 0.423453152179718, -1.8451228141784668, -0.6526002883911133, -0.25456702709198, 0.8700817823410034, -3.6585161685943604, -0.3757256865501404, 0.10140946507453918, 0.44779103994369507, -0.5479230880737305, 1.7670488357543945, -0.81270831823349, 0.3880493640899658, 0.6885210871696472, 2.61816668510437, 0.748733401298523, 1.1726971864700317, -0.46413514018058777, -0.06849156320095062, -0.21761341392993927, -3.000375747680664, 1.2552205324172974, 1.036740779876709, 0.9770087003707886, 0.45604074001312256, -1.7642366886138916, -0.7269617319107056, 1.0188459157943726, -1.888561725616455, 0.22347930073738098, -0.8838891386985779, 1.1890952587127686, -0.6630417108535767, 0.14685536921024323, -0.39034155011177063, -0.565280020236969, 0.755149245262146, -0.15901486575603485, -0.04196203127503395, 0.7160196304321289, -0.1539987474679947, 1.6573290824890137, 0.6189336776733398, 0.1827431619167328, -0.6585545539855957, -0.3729196786880493, 2.403533458709717, 0.9995742440223694, 1.2938282489776611, 0.6250452399253845, -0.488128662109375, 0.12453704327344894, 1.538879156112671, -1.4582709074020386, 0.8774728178977966, -0.32211825251579285, 1.2710940837860107, -1.1827424764633179, 0.059682633727788925, 0.7405533194541931, 0.632567286491394, 0.05423540249466896, -1.0594841241836548, -0.1013757735490799, -1.166748285293579, -0.12636937201023102, -0.44000712037086487, 0.7298099994659424, 0.12396208196878433, -1.439427137374878, 0.2012253701686859, -1.2881224155426025, 0.34604382514953613, -0.43500250577926636, -1.6408997774124146, 0.26599353551864624, -1.3111975193023682, 0.1474376916885376, 1.943983793258667, 0.2331923544406891, -0.3941614329814911, 0.6352546811103821, 0.7864210605621338, 0.8320254683494568, 0.7951154708862305, 0.7703448534011841, -0.6964253187179565, -0.04633992165327072, 3.3226993083953857, 0.6615602970123291, 0.5272900462150574, 0.7611525058746338, 1.1515148878097534, 0.08536411076784134, 0.25012120604515076, 0.10878986865282059, 0.8565364480018616, 1.2595628499984741], [0.8501557111740112, 0.892657458782196, -0.7966403365135193, 0.4474593698978424, 1.3883800506591797, 1.468990683555603, -0.12193681299686432, 1.1138159036636353, -1.2761123180389404, -1.3355640172958374, -0.2092415690422058, -0.1312677264213562, 1.4606965780258179, -0.41255050897598267, -0.121384397149086, -0.3348928987979889, -1.5718212127685547, -1.0097841024398804, -0.24432338774204254, -0.9161823987960815, -0.042440228164196014, 1.4168795347213745, -0.4373973608016968, -1.3364830017089844, -0.4089829623699188, 2.241852283477783, 0.31632721424102783, -1.0607032775878906, -1.0678484439849854, 0.1673911064863205, -1.2628588676452637, 0.7007717490196228, 0.468111515045166, -1.5376392602920532, -0.4802516996860504, 0.9692866802215576, -0.36493685841560364, -0.7028455138206482, 0.07289476692676544, 0.9631690979003906, -0.18415120244026184, 0.5739782452583313, -0.06760776042938232, 1.0860689878463745, 0.10808076709508896, -0.09912801533937454, -1.400238037109375, 0.8716601729393005, -0.26324939727783203, 1.026361346244812, 0.2978830635547638, 1.1496976613998413, -1.409542441368103, -0.22136588394641876, 0.5789060592651367, -1.1946731805801392, 2.9797849655151367, 0.5031569600105286, 0.7335317730903625, -0.17139634490013123, 0.30246207118034363, 0.2115502655506134, -0.5136805772781372, 0.4808991849422455, -1.9150266647338867, 1.057033658027649, -0.49150317907333374, -0.5058261752128601, -0.41150376200675964, -0.5924209356307983, -0.26181504130363464, 0.5941871404647827, -1.3193321228027344, -0.17921534180641174, -0.43850764632225037, 0.51488196849823, 1.0094685554504395, 1.0246776342391968, 1.9519388675689697, 0.11736097931861877, -0.11649402230978012, 2.8582279682159424, 0.27305853366851807, 0.8532057404518127, -0.45615145564079285, -1.2263195514678955, 0.9911553859710693, -0.0498216338455677, 0.03517068549990654, 1.3513562679290771, -0.16583290696144104, 0.919179379940033, -0.2803873121738434, 2.2381250858306885, 0.2750774323940277, 0.05418071895837784, 0.13333922624588013, 0.13639700412750244, 0.20055843889713287, 0.7319758534431458, 0.048152077943086624, 0.4698179066181183, 0.0736389085650444, 1.9984831809997559, -2.103544235229492, 1.1338398456573486, 0.03577728569507599, -0.26578930020332336, 0.5936257243156433, 0.4988126754760742, 1.4718947410583496, 1.0300854444503784, -1.006848931312561, -0.7035118937492371, 0.8621076941490173, 2.1504125595092773, 0.1129385381937027, 0.05249517783522606, 0.19536511600017548, 0.5006822347640991, -0.7115760445594788, -0.19114813208580017, 0.6657635569572449, -1.2375249862670898, -0.19091007113456726, -0.02183418720960617, 2.1340065002441406, -1.1056015491485596], [0.8426752686500549, 1.0168404579162598, -0.16961681842803955, 0.861451268196106, 0.7437964081764221, 0.10859137773513794, 0.90440833568573, 0.805443286895752, 0.8690775036811829, 0.6100926399230957, 1.08948814868927, -0.781792402267456, -1.5468392372131348, -1.2130393981933594, 1.1315197944641113, 0.8575085401535034, -0.8442085981369019, 0.5736598968505859, 1.0005162954330444, 1.4123865365982056, 0.19077026844024658, -1.2847901582717896, -2.025599479675293, -0.00898792315274477, 0.1351415514945984, 1.041855812072754, 0.021508462727069855, -1.8085016012191772, 0.10694847255945206, -0.7338799834251404, -2.3171963691711426, 0.004889234434813261, -0.5031943321228027, 0.6234626770019531, 0.9346479177474976, -0.09962104260921478, -1.1898494958877563, 0.6060958504676819, 0.7150098085403442, -1.3630834817886353, 2.791341781616211, -0.9891680479049683, -0.47793129086494446, -1.8243497610092163, -0.1163317859172821, 1.0097359418869019, 0.1665608286857605, 0.8626463413238525, 0.9027870297431946, -1.1794140338897705, 0.1471838653087616, -1.4572417736053467, -1.0328335762023926, -0.48855581879615784, 0.052780184894800186, 0.31402960419654846, 0.7222388982772827, 0.24681270122528076, -0.13188207149505615, -1.0003082752227783, 1.6031559705734253, 0.33612021803855896, -0.057067565619945526, -0.36460956931114197, -2.1877975463867188, 0.6740939021110535, -2.4882748126983643, -0.9212899208068848, -0.8411148190498352, 0.45256415009498596, -0.1460987776517868, 0.6278048753738403, 0.5400404334068298, 1.154054045677185, -0.3914242386817932, -0.2855869233608246, 2.202827215194702, 0.1697210818529129, -0.4152545928955078, 3.2166335582733154, -1.083120584487915, 0.3597450256347656, -1.5236049890518188, -0.9208166003227234, -1.2259292602539062, 0.5691308379173279, -1.5465675592422485, 0.1737842559814453, -1.955068588256836, -1.625227451324463, 0.4657246768474579, 0.40008246898651123, -0.05296672135591507, 0.19733881950378418, -0.8808001279830933, -0.24411514401435852, -0.07627923786640167, -1.8305000066757202, 0.4201003313064575, -1.392838478088379, -1.5281503200531006, -1.025627613067627, 0.06967748701572418, 0.6709995269775391, 1.7684553861618042, 0.016667036339640617, -2.899254560470581, -0.19052395224571228, 1.0865777730941772, -0.04335368424654007, -2.222947597503662, -0.3692348003387451, -0.5810656547546387, 0.8324219584465027, 1.2196303606033325, 0.34497150778770447, -1.6083557605743408, 0.252223938703537, -1.5541754961013794, -1.0019676685333252, -0.7679868340492249, -1.3817311525344849, 0.2909262478351593, 1.1715766191482544, -1.051906704902649, -1.0715458393096924, 0.228388249874115, 1.061375379562378], [0.5923723578453064, 0.04922720789909363, -0.47903361916542053, 0.34058669209480286, 2.1563780307769775, -0.5221576690673828, 1.5213189125061035, 0.25477296113967896, -0.4475969076156616, 1.243699312210083, -1.3171372413635254, 0.8242641687393188, -1.000178575515747, 0.02332502231001854, -1.6119977235794067, -0.736923098564148, -0.2729949355125427, -0.6827676892280579, 0.5848673582077026, -0.7656503915786743, 0.03747878968715668, -1.3674589395523071, -0.3368425965309143, 1.274827003479004, -1.7066045999526978, 0.13905058801174164, -0.04504282400012016, -1.4650042057037354, 0.13947874307632446, 0.08537232130765915, 0.06242768466472626, 2.1688547134399414, -1.7031911611557007, -1.3409394025802612, 1.363987684249878, -0.9818518757820129, -1.2667783498764038, -0.2151114046573639, 0.16953079402446747, -0.8671852946281433, 1.500654697418213, -1.6475571393966675, -0.7523136734962463, -0.7036875486373901, 0.9001194834709167, -0.046477507799863815, 0.027644114568829536, -0.29762277007102966, -0.27216315269470215, 0.9632137417793274, -0.38622504472732544, -1.124860167503357, 0.3130963444709778, 0.36006197333335876, -0.06731638312339783, -0.5440850257873535, 1.265921950340271, 0.4703120291233063, -1.000304937362671, -0.8657282590866089, 0.6782717108726501, 0.44644680619239807, 2.4028420448303223, 1.621418833732605, 0.5127443671226501, -0.5086431503295898, -0.4720989465713501, -0.5157191753387451, 0.16295146942138672, -1.256471037864685, 0.01689731329679489, -0.6653075218200684, 1.3107789754867554, 1.9734785556793213, 0.35986268520355225, 0.2625684142112732, 0.05276274308562279, 0.8006479144096375, -0.7305464148521423, -0.7378854751586914, -1.0999904870986938, 0.23819351196289062, -0.23050309717655182, -1.0290857553482056, -1.1461000442504883, -0.3465537428855896, 0.42408287525177, -1.658492088317871, 0.5529453754425049, 0.928936243057251, -0.5021100044250488, -1.1545475721359253, -0.9965564012527466, -0.7431314587593079, -1.5824856758117676, 0.210488423705101, -0.41271916031837463, 0.9785224199295044, 0.21921513974666595, 1.1036535501480103, -0.9048159122467041, -1.03175950050354, 0.6851317286491394, 0.16375333070755005, 0.14986015856266022, -0.060302991420030594, -0.7479629516601562, 0.13946130871772766, -0.6421606540679932, 0.07987148314714432, -0.02792675979435444, -1.7535903453826904, -0.9225027561187744, 0.6313111186027527, 0.5980545878410339, -0.10365397483110428, -2.100233554840088, -0.5776396989822388, -0.39468613266944885, -0.011027482338249683, -1.1272871494293213, 0.7200561761856079, 1.4393348693847656, 1.0607060194015503, 0.09326063841581345, 0.378926545381546, 0.9441676735877991, 1.9497005939483643], [-1.6742843389511108, 0.33345678448677063, -0.2610170245170593, -0.08362148702144623, 1.8131444454193115, 0.4636351466178894, -0.31682831048965454, 0.48554155230522156, 0.40405967831611633, -0.06878092885017395, 0.06249670311808586, -1.207471251487732, -0.8819466829299927, -0.7549971342086792, -1.3285249471664429, 0.07140705734491348, 0.27626967430114746, -0.9086031913757324, 2.569629669189453, -1.4500709772109985, -0.9430503249168396, -0.5092523097991943, 2.062012195587158, -0.7147800922393799, -1.6568551063537598, 0.8550699353218079, 1.741751790046692, 0.5833393335342407, -0.4438130259513855, 0.3398340940475464, 0.5249106287956238, 0.13179339468479156, 0.566074013710022, 0.5729885697364807, 0.37067651748657227, -0.1171613559126854, 1.4084255695343018, 1.1276010274887085, 0.9584436416625977, 0.9910212755203247, 1.222922921180725, 0.38036102056503296, 1.4355896711349487, -2.032625675201416, 1.6193437576293945, -0.2652475833892822, 1.348266363143921, -0.2504516541957855, 0.3746354281902313, 0.7369597554206848, -1.6023582220077515, 0.4501628875732422, 0.3718690276145935, 1.9786655902862549, 0.04086661711335182, -1.8577498197555542, -0.01533471792936325, -0.9605995416641235, -0.00547730503603816, -0.10485329478979111, -0.5776455402374268, -1.2365469932556152, -1.0651475191116333, -1.5431331396102905, -1.3306795358657837, -0.15187746286392212, -1.1917471885681152, 1.015268087387085, -0.05578886345028877, -0.45679986476898193, 0.2870934307575226, -0.42018067836761475, 1.3441492319107056, 1.795337438583374, -1.055272102355957, -0.2829974889755249, 0.725337564945221, 1.9022374153137207, -0.380327433347702, 2.9092726707458496, -0.49982112646102905, 0.9902332425117493, 0.9038835167884827, 0.5022275447845459, -1.3663864135742188, -1.4657913446426392, 0.7405619025230408, 0.7538650631904602, -1.1972572803497314, -0.592284619808197, -0.3937135934829712, 1.0413607358932495, -0.7310154438018799, -1.3685178756713867, -0.8662355542182922, -0.21307754516601562, -1.5566290616989136, -2.353341817855835, 1.0140182971954346, 1.1081382036209106, 1.389572024345398, -0.19305147230625153, 1.1077876091003418, -1.3164405822753906, -0.3379209339618683, 0.6148125529289246, -0.7215631008148193, -0.0008052410557866096, -1.6345736980438232, 0.08679813891649246, -0.9906057119369507, 1.435287594795227, 0.15943971276283264, 1.0742394924163818, -0.36445116996765137, 0.8683596253395081, 0.3981192111968994, 0.5784909129142761, 0.17393054068088531, -1.0290783643722534, 0.2196233868598938, 0.9224875569343567, 0.8056093454360962, 0.36468324065208435, -0.6762480139732361, 0.6180616021156311, 0.38421207666397095, -0.5026047229766846], [0.4226894974708557, -1.2392505407333374, 0.03921445831656456, -2.0131571292877197, 1.4982987642288208, 0.06755964457988739, -0.13072265684604645, -0.33453890681266785, 2.3880774974823, -0.5421101450920105, 1.9333882331848145, 0.06329987943172455, 0.39197856187820435, -1.0850485563278198, 1.977940559387207, -0.6453419923782349, -1.1900074481964111, 0.09293317049741745, -1.4154282808303833, -0.48707473278045654, -1.544190764427185, -2.1150710582733154, 1.945779800415039, -0.2177548110485077, 0.22121141850948334, 1.0197746753692627, -1.350954532623291, -0.32263991236686707, -0.0417868010699749, 0.3142179846763611, 1.813781976699829, 0.8064289093017578, -0.9578153491020203, -0.9615060687065125, 0.5639283061027527, -1.0991419553756714, -1.3317110538482666, -0.8165816068649292, -0.6532730460166931, 0.3198326826095581, 0.27239951491355896, 0.10621923953294754, -0.6082231998443604, 0.8618195652961731, -0.8908470273017883, 1.1114346981048584, 1.1275649070739746, 0.2503774166107178, 0.29362767934799194, 0.2685810327529907, -1.2318918704986572, -0.41088756918907166, -0.44927623867988586, -0.8046176433563232, 1.6981582641601562, -0.014504033140838146, 0.1360052525997162, -0.6799917221069336, -0.08659003674983978, -1.3088712692260742, -1.7734590768814087, 0.6765496730804443, -0.93623948097229, -1.2433704137802124, -0.24947290122509003, -0.5236043334007263, 2.3151464462280273, 0.715968132019043, -1.3624186515808105, 0.8782845735549927, 0.025184836238622665, 1.28142511844635, 0.5459899306297302, 0.5735476016998291, -2.6896986961364746, 1.030311107635498, 0.5481778979301453, 0.38730087876319885, -0.8078581094741821, -1.4532634019851685, -1.0236377716064453, 0.08003249764442444, 0.7485021352767944, -1.3966896533966064, -0.15408411622047424, -0.652478039264679, 0.6798706650733948, -0.6094809770584106, 3.1276440620422363, 1.309618592262268, 2.175211191177368, 0.873388946056366, 0.3945028781890869, 0.3457320034503937, -0.30492493510246277, -0.8233516812324524, -0.6709299087524414, -0.15414594113826752, -0.0034155172761529684, 0.7790625691413879, 0.11865229159593582, -2.5265071392059326, -1.3713433742523193, 1.7371950149536133, 0.9024174809455872, 0.9049058556556702, -0.7300807237625122, -0.3913700580596924, -0.1806073784828186, 0.01299399882555008, 2.0885462760925293, -0.08142103254795074, 0.5904832482337952, -0.5790114998817444, 0.22176383435726166, -0.7903265357017517, 0.033546559512615204, 0.9004386067390442, 0.8399771451950073, -1.634169101715088, 0.47260522842407227, -0.051850154995918274, -1.3104881048202515, 1.2225332260131836, -1.0053311586380005, -1.4593515396118164, -0.4802454113960266, 0.5864952206611633], [2.2731306552886963, -1.8200918436050415, 0.40115833282470703, 1.3841639757156372, -1.1624538898468018, 0.19009609520435333, 0.45757076144218445, -1.0162546634674072, -0.3458387553691864, -0.539688766002655, 0.6036487817764282, -0.17303742468357086, -0.7844653129577637, -0.030866088345646858, -1.769029140472412, 1.3034307956695557, -0.8524297475814819, 0.9623281955718994, -1.244852900505066, 1.0083913803100586, -0.39077436923980713, -0.2477007955312729, -0.0832776129245758, -0.5627114176750183, 0.7826903462409973, -1.2383646965026855, 1.0397454500198364, 0.26629024744033813, 0.010144788771867752, 0.4101322591304779, 0.6877068281173706, 0.21599289774894714, -0.5326271057128906, -0.9191762804985046, -1.5969159603118896, 0.7466812133789062, -1.0962660312652588, -2.0920681953430176, 2.3373823165893555, 0.4499821364879608, 0.016702355816960335, -0.014964446425437927, 0.18638557195663452, 1.2004882097244263, -0.8847586512565613, 0.3866957724094391, -0.01712390035390854, 0.5536295175552368, 1.5620672702789307, 0.015398822724819183, -1.1006507873535156, -0.543886661529541, -1.0250751972198486, -0.22481054067611694, 0.21464979648590088, 2.0597121715545654, -0.20681071281433105, -0.1908567100763321, -1.8172309398651123, 0.4364983141422272, -1.0393086671829224, 0.13381153345108032, -1.2840162515640259, -0.25320467352867126, -0.23240137100219727, -0.19875483214855194, 0.9101184606552124, -0.7713057994842529, -0.01845174841582775, 0.8730937242507935, 1.6830990314483643, -0.6085613369941711, 0.10648978501558304, -0.3946684002876282, 0.8436065316200256, -1.4416439533233643, 0.43405503034591675, -0.2549724876880646, -1.1158658266067505, 0.4872782528400421, 0.5463345050811768, -0.5791986584663391, 0.26026564836502075, -1.76421058177948, 2.490516424179077, -0.6347703337669373, -0.17852239310741425, -1.0478003025054932, -0.9610781669616699, 0.6896606087684631, -0.9095737338066101, 1.4050935506820679, 0.7327061295509338, 1.0723744630813599, -0.29835814237594604, -0.010922136716544628, -0.284086674451828, 0.8381834626197815, -0.025348659604787827, 1.8516913652420044, -1.2166118621826172, 0.027059374377131462, 1.582943320274353, 1.8763480186462402, -1.262887954711914, 0.7324825525283813, -0.5181363224983215, 1.0186975002288818, -1.552828073501587, 0.02866843156516552, 0.7603630423545837, 0.4059164524078369, 0.31879714131355286, 0.7588362693786621, -1.4468668699264526, 0.20828257501125336, 1.1705310344696045, -1.5086719989776611, 0.6149861812591553, 0.9156225919723511, -0.4153459966182709, -0.03533705696463585, 1.1534581184387207, -0.14303991198539734, 1.0211880207061768, -0.2038547694683075, 0.19630900025367737, 0.2569524943828583], [-1.5853447914123535, -1.138922929763794, 0.8175270557403564, 0.6564183235168457, -0.43863463401794434, -0.22102488577365875, 0.4574880003929138, 0.9233343005180359, 0.6991154551506042, -0.9587454199790955, 0.675488293170929, -1.510863184928894, -0.557830810546875, -0.7352266311645508, 1.084705114364624, 1.3532546758651733, -1.1249468326568604, 2.2101664543151855, -1.7734804153442383, -0.053787458688020706, 0.15820874273777008, 0.7763722538948059, 1.2439786195755005, 0.1510372757911682, -0.488335520029068, 0.7070015072822571, -0.7284973859786987, -0.39815202355384827, 0.5039201974868774, -1.0854977369308472, -0.2676308751106262, 0.10662015527486801, -0.07977431267499924, 0.08161891251802444, -0.703135073184967, -0.27372509241104126, -0.2728586494922638, 1.164154291152954, -0.7814605236053467, -0.5851203203201294, -1.447749376296997, -0.4673168659210205, -0.6768014430999756, -1.172096610069275, -0.3432503938674927, -1.4019572734832764, 0.7737759947776794, -0.7542235255241394, -1.480786681175232, -1.2011469602584839, 1.1514863967895508, -0.7789342403411865, 0.4130809009075165, 1.971583366394043, -0.31606048345565796, -0.2965405583381653, 2.564147710800171, -2.9474411010742188, 0.2629481256008148, -0.3675880432128906, 0.6952161192893982, 0.07045569270849228, -0.4419032335281372, -0.9673005938529968, 0.6826059818267822, -1.1911803483963013, 0.9737033843994141, -1.8618601560592651, 1.015428066253662, 0.8173711895942688, 0.0630280077457428, 0.43003180623054504, -0.2563991844654083, 2.1624491214752197, -0.6090916395187378, -0.986664354801178, 0.33604151010513306, 0.8404337763786316, -0.7295275330543518, -1.2301548719406128, 1.2888797521591187, 1.4513503313064575, -1.4314627647399902, 0.047509465366601944, 0.5933550596237183, 0.7952526807785034, -1.5306366682052612, -1.4766825437545776, 0.4054640233516693, -1.2896497249603271, 0.42885294556617737, 0.3224762976169586, -0.6405027508735657, 0.5814828276634216, 0.2258383184671402, -0.3983345925807953, -0.3493511974811554, 0.7733845710754395, -0.1382126808166504, -0.8325175046920776, 0.10973560810089111, -0.105631023645401, -0.4773554801940918, 0.16705144941806793, 2.5825912952423096, 0.9722303748130798, -0.6389495730400085, 0.3894980251789093, -1.5902615785598755, 0.4380728304386139, 0.17962989211082458, 2.3060364723205566, -1.4881247282028198, -0.8912052512168884, 0.39876559376716614, -0.658020555973053, 0.31445014476776123, -0.319977730512619, 2.0224061012268066, -0.20077091455459595, -2.1228578090667725, 3.388862371444702, -0.5593507885932922, 0.5114235281944275, -1.1255029439926147, 0.15273191034793854, -1.9938162565231323, 1.192213535308838], [-0.13427521288394928, -2.488255500793457, -0.9446689486503601, 0.8248354196548462, 0.35761117935180664, 1.5643126964569092, 0.12878401577472687, -0.3698938190937042, 1.8494445085525513, -0.39410749077796936, 0.6852530241012573, 0.2688681185245514, 0.5627680420875549, -1.9521414041519165, 0.060183051973581314, -0.09399549663066864, -0.809256374835968, -1.026408314704895, -1.5366542339324951, -0.7948126196861267, -0.9296280741691589, 1.1126364469528198, -0.3150644302368164, 0.8832636475563049, -0.8336715698242188, 0.023156175389885902, -1.3503273725509644, -1.3255733251571655, 0.2144014984369278, 0.3774377405643463, 0.8187999129295349, 1.7483563423156738, -1.0331480503082275, 0.1341262310743332, -1.1094112396240234, 0.4051620066165924, -0.20895268023014069, 0.010137125849723816, -0.7065840363502502, 0.8548064231872559, 0.4253063201904297, 1.1017807722091675, 1.14945387840271, -1.0077121257781982, -1.6375316381454468, -0.2837982773780823, 0.7433239817619324, 1.1520240306854248, -0.41825032234191895, -1.1941041946411133, -0.17492976784706116, -0.9936503767967224, -0.5447920560836792, -1.949087142944336, -1.1443376541137695, -0.5092299580574036, -2.8769049644470215, -1.233234167098999, 0.5648033022880554, -2.098393201828003, 0.24394460022449493, 1.4155433177947998, 0.033860161900520325, -0.021002735942602158, 0.6829107999801636, -1.0505821704864502, 2.115774631500244, -0.12911579012870789, -2.030989408493042, 0.9988698959350586, 0.9124969244003296, -1.2615638971328735, 0.3040173053741455, 1.2394216060638428, -1.3248943090438843, 0.2925369441509247, 0.17862951755523682, 0.9868144989013672, 1.0211398601531982, -0.3926145136356354, -0.3433251976966858, 0.6819328665733337, -0.20370879769325256, 1.0288398265838623, 0.05323242396116257, -1.252204179763794, -1.9196455478668213, 0.24246160686016083, 0.10436505824327469, 0.6821143627166748, 0.002224238822236657, -0.7243969440460205, -0.32231083512306213, 1.1369136571884155, 0.4839943051338196, -2.177316665649414, 0.07461658120155334, -0.07833940535783768, 1.3880531787872314, 0.6987531781196594, 0.05950223281979561, 0.8047723770141602, -0.5859618186950684, -1.5448607206344604, 0.8133171200752258, -0.2582966685295105, 0.16580650210380554, 1.4786797761917114, 2.1697957515716553, -0.9471113681793213, 0.01018926315009594, -0.5117943286895752, 1.2546043395996094, 1.1185439825057983, -0.05668400600552559, -0.2380920797586441, -0.33781251311302185, -0.4491254985332489, 0.03169824182987213, 0.050143443048000336, -0.5661391615867615, -1.206616759300232, 1.1954275369644165, -0.7977215051651001, -0.8304976224899292, 1.2897522449493408, -0.27887967228889465, -1.3881323337554932], [-0.39816775918006897, -0.849320650100708, 0.8414976000785828, 2.098839521408081, -0.12460994720458984, 0.5370938777923584, -0.7095720767974854, -1.2264683246612549, -0.42764344811439514, 0.7801768779754639, 1.5673774480819702, 1.5305590629577637, 0.22669728100299835, -0.5828291177749634, 2.002678632736206, 0.5253174304962158, 0.9385426640510559, 0.38539719581604004, -2.0780556201934814, -0.3050336241722107, 0.039529234170913696, 0.8700768351554871, -0.461709588766098, -0.7222170829772949, 1.279605507850647, 0.4433504641056061, -1.0816447734832764, -0.14330436289310455, 0.9716112017631531, -0.07251255959272385, 2.969219446182251, 1.2774500846862793, 0.7318170070648193, 0.4627310335636139, -0.18781137466430664, -1.163831353187561, -0.7399178147315979, -1.6000555753707886, -1.3057518005371094, -2.3739230632781982, -0.3964286744594574, 1.1944491863250732, -1.4067344665527344, -0.28580713272094727, -0.6470693349838257, 0.8736305236816406, -0.06918126344680786, -1.4539422988891602, 0.24300408363342285, -0.8501731157302856, 0.1059015616774559, 0.10583601146936417, -0.21552689373493195, 0.4250901937484741, -0.10084354132413864, 0.061429280787706375, -0.06581419706344604, 2.2649662494659424, 1.410439372062683, -0.33622944355010986, 0.316849946975708, -2.0786943435668945, -0.7840953469276428, -0.7776125073432922, -0.015456503257155418, 0.1043367013335228, -0.27672332525253296, 0.7741432189941406, 1.3498826026916504, 0.5815457701683044, 0.14102216064929962, -2.022839307785034, -0.11508037894964218, 1.5725913047790527, -2.5925745964050293, -0.6693695783615112, -0.7526468634605408, -0.2548757493495941, -1.4466373920440674, 0.5951774716377258, 0.5952891111373901, -0.1672644466161728, 1.1334940195083618, 0.45997554063796997, -0.8544591069221497, 0.1401507705450058, 1.4629932641983032, 0.5273640155792236, 1.4851300716400146, -0.9982955455780029, 1.6703702211380005, 0.30329832434654236, -0.31725946068763733, 0.2964245676994324, 0.987092912197113, -0.566332221031189, 2.2341439723968506, 0.6546227931976318, -0.7128633260726929, -1.6095434427261353, 2.1755707263946533, -0.5261672735214233, 1.3617178201675415, -0.14012350142002106, -0.04829215258359909, -0.5233706831932068, 0.6167672276496887, -0.332250714302063, -0.7031364440917969, -0.4012520909309387, -1.3763806819915771, 0.29796913266181946, 0.5108318328857422, 0.9496219754219055, 0.36471131443977356, 0.5025008916854858, -1.6505968570709229, 0.5214338302612305, -0.5037803649902344, 0.45566481351852417, -0.09024263918399811, 0.17979919910430908, 0.5218713879585266, -0.4565061330795288, -0.9370886087417603, -1.6470407247543335, -0.6794797778129578, -1.1741845607757568], [-0.6863161325454712, -0.021700862795114517, 0.0036106056068092585, 2.239511489868164, -0.07535591721534729, 0.20591622591018677, -0.19648267328739166, -1.9488716125488281, 0.5183541774749756, -1.140721321105957, -0.9710555076599121, 0.6571006178855896, -0.2671258747577667, 0.5487292408943176, -1.7726083993911743, 0.050362635403871536, 0.6649503707885742, 0.8765999674797058, 0.4212053418159485, -0.010817183181643486, 0.9766098260879517, 0.8991041779518127, -1.358541488647461, -0.42628014087677, -0.40880414843559265, -0.6499050855636597, 0.20707134902477264, -0.05669603869318962, 1.0538005828857422, 1.4908251762390137, 1.494193434715271, -0.2211911380290985, 0.9678504467010498, 0.5658203959465027, -0.20149964094161987, -0.5015381574630737, 0.8703970909118652, 1.894789218902588, -1.1221531629562378, -0.34775498509407043, 0.33576157689094543, 0.5628989934921265, -0.7988929152488708, 1.0789715051651, 0.45142316818237305, 0.11252794414758682, 0.3927858769893646, 1.7177584171295166, -0.2836470603942871, 0.4480361044406891, -0.646460771560669, -0.9122582674026489, 0.436991423368454, -0.49270153045654297, -0.6102678179740906, 1.1857917308807373, -1.150403618812561, 0.8462603092193604, -0.22206807136535645, 0.020692894235253334, -0.7317925095558167, -0.6211466789245605, -0.025716004893183708, -0.10205067694187164, -0.6265857219696045, -0.33376815915107727, -0.3868946433067322, -0.06713210791349411, -1.6701136827468872, 0.7656205296516418, -1.1339083909988403, 0.4726611077785492, 0.45040255784988403, 0.9887039661407471, -2.161024808883667, -0.49678879976272583, -1.8777601718902588, -0.5913706421852112, 0.1589280664920807, 1.1400922536849976, 0.7695087790489197, 0.06250006705522537, 0.9171642661094666, -0.5148950219154358, 0.5548708438873291, 0.3824104368686676, 0.04703545942902565, 1.1969764232635498, 0.7296481132507324, -0.7196521162986755, 0.206041619181633, 1.1398513317108154, 1.0356656312942505, 0.4333835542201996, -0.9897101521492004, 0.37752506136894226, -0.42153531312942505, 1.3778973817825317, -1.2679373025894165, 1.262074589729309, -0.22536517679691315, -1.2761515378952026, -2.1908817291259766, 0.8301591873168945, -0.1289365440607071, 0.30691495537757874, 0.42206019163131714, 0.7420799136161804, -2.8319122791290283, 2.3600223064422607, -0.7041349411010742, 0.8941081166267395, 0.2991653382778168, 1.8705637454986572, 0.2985823154449463, 0.624316930770874, -0.8784590363502502, -1.0280957221984863, 2.3089187145233154, 0.3470971882343292, 0.02226690948009491, 1.5679458379745483, -0.3593596816062927, 1.8859163522720337, -1.0327335596084595, 1.2932730913162231, -0.615908682346344, 0.20397014915943146], [-0.09379197657108307, -1.9962724447250366, 0.4852420389652252, -0.1755007654428482, 0.835622251033783, 0.8454358577728271, -1.6879476308822632, -0.5927635431289673, 0.12542565166950226, 0.8858105540275574, 0.28362104296684265, -1.2422016859054565, -0.7383767366409302, 1.3007224798202515, 1.335720419883728, 0.5599468350410461, 0.5923939943313599, -0.6752468347549438, 0.7869361042976379, 1.3012747764587402, -3.203237771987915, 1.8435481786727905, 0.6395443081855774, 1.0216476917266846, 0.9727802872657776, 1.6482821702957153, -0.37739402055740356, 0.16983076930046082, 1.1774457693099976, -0.6941146850585938, -0.23756223917007446, 0.6269533634185791, 1.3157665729522705, -0.5274592638015747, -1.7032431364059448, -1.1688110828399658, -0.5192530155181885, 1.635398268699646, -0.09292635321617126, -0.7362163662910461, -0.5585469007492065, 0.9580547213554382, -1.759476661682129, -0.15172983705997467, 1.556337833404541, 0.3121704161167145, -1.275650143623352, 0.7143244743347168, 0.32555651664733887, 1.5471410751342773, 1.2855989933013916, -1.5267658233642578, -0.3477105498313904, 1.7959555387496948, -0.21370598673820496, 1.2310082912445068, -0.06757145375013351, -0.6780627965927124, -0.29660478234291077, -1.975098729133606, 0.6330953240394592, -0.8134247660636902, 0.27396708726882935, -0.14131225645542145, 0.9129247665405273, -1.1480342149734497, -0.3047024607658386, -0.5717043280601501, -0.7499653697013855, 0.49007448554039, 1.6845905780792236, -0.10517869889736176, -0.46083545684814453, -1.9889216423034668, -0.026667168363928795, 0.7675496339797974, 0.2194749414920807, 1.1821492910385132, 0.45800521969795227, 0.4403729736804962, -2.351790189743042, 0.04289468750357628, 0.17814457416534424, 0.0028596112970262766, -0.7443169951438904, 1.1541211605072021, 0.09270833432674408, -1.4914225339889526, 0.9435726404190063, -0.3575683832168579, -0.5981990098953247, -1.6703181266784668, 0.4457336664199829, -0.007918172515928745, 0.9259700775146484, -0.8058423399925232, 0.010323366150259972, -0.013198466040194035, 0.4141002595424652, 0.824140191078186, 1.1332215070724487, -0.25074082612991333, 0.8497443795204163, 0.3573383092880249, 1.9920235872268677, 3.6708974838256836, 2.301109552383423, 0.7281401753425598, -0.6842353940010071, -1.3626073598861694, 1.1034801006317139, -1.5692020654678345, 1.762819766998291, -0.7316243648529053, 2.306032419204712, 0.9209544062614441, -1.354882836341858, 0.47861114144325256, 0.09605682641267776, -1.3426491022109985, -0.22557437419891357, 1.6058179140090942, 0.7172946929931641, 0.4558175802230835, 0.2295762598514557, -0.8795448541641235, -0.035411275923252106, -0.6628972291946411], [-0.8465783596038818, 0.24945765733718872, -0.048065874725580215, -0.16271238029003143, -0.9329465627670288, -1.50546395778656, -1.0274535417556763, -1.7452492713928223, -0.1862277090549469, -1.2039557695388794, -0.09704840928316116, 0.12952877581119537, -0.8579588532447815, 0.7488943934440613, 0.1974693089723587, -1.8467317819595337, -0.6272196173667908, 1.9912629127502441, -1.8595236539840698, 0.9406784772872925, -1.8933404684066772, 0.0006738129304721951, -0.7712734341621399, 1.0977615118026733, -0.7713631987571716, -0.7689166069030762, -1.07658851146698, -0.37754297256469727, 1.9560024738311768, -0.48359137773513794, -0.7872973680496216, -1.0149980783462524, 0.30431365966796875, -1.9439520835876465, 0.1640835702419281, 1.973632574081421, -1.5669431686401367, 1.2765884399414062, -1.3604516983032227, -0.9313194751739502, 0.585989773273468, -0.6315059065818787, -1.5710773468017578, -0.28693071007728577, -0.46974262595176697, 0.32769638299942017, -0.3454076051712036, -1.6719568967819214, 0.42452889680862427, -0.13848334550857544, -0.5476919412612915, 1.3024789094924927, -0.7762354612350464, 0.9347951412200928, -1.8446663618087769, 1.8347359895706177, 0.3440703749656677, 0.6740133166313171, -1.2006558179855347, 0.3029960095882416, -2.156404972076416, -2.261711835861206, -0.3474273383617401, 1.5284382104873657, 1.997449517250061, 0.058534979820251465, -1.5007675886154175, -1.418161153793335, 0.820441484451294, 0.45278412103652954, -1.0277142524719238, -0.4529785215854645, 1.444018840789795, 0.6619062423706055, 0.0020270184613764286, -0.5482857823371887, -0.2472078651189804, -0.70097416639328, -1.0118986368179321, 0.2145083248615265, -1.5016485452651978, -0.24583405256271362, -1.774051308631897, 0.13164615631103516, -0.9821722507476807, -0.018052970990538597, -0.7760834693908691, -0.7935580015182495, 1.3284268379211426, 0.8747313022613525, 1.2285817861557007, 0.4539274573326111, 0.21644224226474762, -0.31615185737609863, 0.7472224235534668, -0.7760223150253296, 0.10025472939014435, -1.5554007291793823, -2.093536138534546, -0.43968093395233154, 0.3808366060256958, 1.7601546049118042, -0.7605502605438232, -0.8153784275054932, -0.40336692333221436, 1.2422981262207031, -0.524910032749176, -1.7048438787460327, 0.9412360787391663, 0.2509426772594452, -0.0528528057038784, 0.5206910967826843, -0.7987545728683472, -0.8904469609260559, 0.019277483224868774, -1.8594359159469604, -1.1499747037887573, 1.7308554649353027, -1.260978102684021, 1.174919843673706, 0.9518308639526367, -0.6649860739707947, 1.7360767126083374, 0.9505337476730347, 0.5832464098930359, -2.3668386936187744, 0.12669618427753448, -1.6614000797271729], [0.3601112961769104, -1.1717687845230103, -0.477131724357605, -0.2936362624168396, -0.9471578001976013, 0.1616765558719635, 2.3883416652679443, -0.1354706734418869, 0.039187636226415634, 0.8468939065933228, 0.224545419216156, -0.134109228849411, 0.9447107911109924, 0.7764072418212891, 1.0801286697387695, 0.3350701630115509, 0.18133501708507538, -0.9406602382659912, 2.2844743728637695, 0.642258882522583, 0.28954586386680603, 1.1975756883621216, -1.3091013431549072, -0.3640930950641632, 0.09708569943904877, -0.5056155920028687, 0.18222743272781372, -0.05597569793462753, -1.2773103713989258, 1.4349323511123657, -0.21217729151248932, 0.25705215334892273, 0.5949111580848694, 0.39105531573295593, -0.08136176317930222, 1.2677114009857178, -0.10271012783050537, -0.47234398126602173, 0.03608211502432823, -1.004886507987976, -2.48958420753479, 0.5853361487388611, -1.6785424947738647, -0.06460748612880707, -0.2133025974035263, -1.9145615100860596, 1.902742624282837, -2.263813018798828, -0.045523449778556824, 0.30732572078704834, -0.10490109771490097, 1.2985811233520508, -0.806641697883606, 0.479373574256897, 0.44265156984329224, -0.05974826589226723, 1.7291184663772583, 0.6648095846176147, -0.9263502955436707, 2.1105546951293945, -1.2967674732208252, -1.0469486713409424, 0.05638578161597252, -1.893907904624939, -0.8261733055114746, -1.54417884349823, 0.9808646440505981, 1.069329857826233, -0.03528639301657677, 0.2840830683708191, -0.889116644859314, 0.47788873314857483, -0.33163055777549744, 1.4744642972946167, 0.36948180198669434, -0.7050499320030212, -0.23137152194976807, 0.855002760887146, 0.9579254984855652, 0.9432278275489807, 0.7416967749595642, -0.22306139767169952, -0.33007776737213135, 1.582944631576538, 0.6047495603561401, 0.14844778180122375, 1.0138959884643555, -0.5454587340354919, -0.7347758412361145, 0.12027478963136673, 0.5618093013763428, 0.9684557914733887, 0.3577098846435547, 0.5332005023956299, -0.9751952886581421, -0.8870190382003784, -0.23523865640163422, -0.6409240961074829, -2.0343544483184814, 0.29156404733657837, -0.2940213978290558, -0.7295416593551636, 0.04358135908842087, -0.7889358401298523, 1.042988896369934, 0.4948394298553467, 0.5875096917152405, -0.2660442888736725, -0.260845422744751, 0.9420420527458191, -1.8525395393371582, -0.4361816942691803, 1.2037557363510132, -0.2658795416355133, -1.3686293363571167, -0.913016140460968, 0.8017066717147827, 2.357081890106201, -1.5741852521896362, -0.011037669144570827, -1.3713434934616089, -0.4725058674812317, -0.17330287396907806, 0.5187586545944214, -0.6825974583625793, -1.0152028799057007, -0.11502595990896225, -2.292706251144409], [-1.2661347389221191, -0.18478858470916748, 0.9615389108657837, 0.06698257476091385, 1.6129900217056274, -1.4127836227416992, -0.3237355351448059, -0.9411302208900452, -0.7020367383956909, -0.3010305166244507, 0.3383689224720001, 0.656032145023346, 0.9874903559684753, -1.0900964736938477, 0.975756049156189, 0.018958067521452904, -1.2633548974990845, 0.7235466241836548, -0.39963576197624207, -1.2380714416503906, 0.521102786064148, -0.29251599311828613, -0.7767835259437561, -0.3592793643474579, -0.4728417992591858, 1.2142106294631958, -0.22438813745975494, -0.057474177330732346, -0.7110173106193542, 1.2941672801971436, 0.037555817514657974, 0.04906583949923515, -0.6291754841804504, 1.043479561805725, 0.4585361182689667, 0.619217574596405, 0.17620237171649933, -0.8570483922958374, -0.9070245623588562, -0.3236912488937378, -0.4738878011703491, 0.9825176000595093, 1.3811721801757812, -0.5542489290237427, 2.289698600769043, -0.1523250937461853, 0.151119664311409, -1.869342565536499, -1.2508857250213623, 0.9131284952163696, 1.7629269361495972, -0.15418407320976257, 0.09835685044527054, 1.2944819927215576, -0.9247165322303772, -0.8144465088844299, -0.5163939595222473, 0.4582059979438782, 1.0850303173065186, 2.026435136795044, -0.636783242225647, 0.38057607412338257, 1.8491915464401245, 0.3215418756008148, 1.6216297149658203, -0.10590231418609619, 0.9402409791946411, -0.6235945820808411, -0.7620998620986938, -0.8528894186019897, -0.3151451349258423, -0.2721370756626129, -0.1484392136335373, -0.9271296262741089, -0.3386197090148926, 0.11002840101718903, -1.188326120376587, -0.04137788712978363, 0.3405037522315979, 0.2745721638202667, -0.7223395109176636, 1.5615580081939697, 0.6010091304779053, 0.4147796332836151, 0.15285328030586243, 0.16906534135341644, 0.40314650535583496, -2.2537124156951904, -0.41341927647590637, -0.5335128903388977, 1.3439759016036987, -0.4054950177669525, 2.9203455448150635, -1.2144007682800293, -0.25833165645599365, -1.2553802728652954, -0.2806761860847473, -1.8215667009353638, -0.7340378165245056, -1.0343159437179565, -0.32135531306266785, 0.5569586753845215, -1.8504523038864136, 1.6437420845031738, -0.4225786030292511, -1.8060818910598755, -0.7122315168380737, 0.42280113697052, -0.44362249970436096, -1.6653517484664917, -0.23112644255161285, -0.36446791887283325, -0.5438124537467957, 1.2392386198043823, 2.3471014499664307, 1.6220643520355225, -1.6830518245697021, 0.9030996561050415, -0.5211799740791321, 0.5153630375862122, -0.9264425039291382, 0.2602972686290741, -0.24696922302246094, -1.2664051055908203, 0.30977964401245117, -0.6236593723297119, 0.43303754925727844, 0.5186185240745544], [-1.6051288843154907, -0.10679648071527481, 0.8155761957168579, -2.0844593048095703, -0.9324349164962769, 0.5359466671943665, -0.6645286083221436, 0.7122296094894409, -0.27309831976890564, 1.105035424232483, -1.809510350227356, 1.4582375288009644, -0.003297551302239299, -2.059328317642212, 0.855501115322113, -1.0804756879806519, 2.963817834854126, -0.732227087020874, 1.0847303867340088, -1.503359317779541, -0.7616763710975647, -0.5010174512863159, -0.0634661540389061, -1.6681276559829712, -1.0294655561447144, -1.2968308925628662, -0.8306988477706909, 2.547710418701172, 0.34122434258461, -0.9795224666595459, -0.09182730317115784, 0.6521453261375427, 1.0413398742675781, -0.4675960838794708, -0.4442271292209625, -0.6807997822761536, 1.420650839805603, 0.043629709631204605, 0.45149102807044983, 0.8422252535820007, -2.909269332885742, 0.3515825867652893, 1.7209491729736328, -0.8885916471481323, -0.7677984833717346, -0.5089592337608337, 1.5088133811950684, 1.317800760269165, 0.5476685166358948, -0.5817367434501648, 1.2402281761169434, -0.618073046207428, 0.2001921534538269, 0.9287322759628296, -0.5003971457481384, -1.048296570777893, 0.4446294903755188, 0.8445277810096741, -0.46920841932296753, 0.04112778604030609, 0.02954833395779133, -1.8339664936065674, 2.4129292964935303, -0.10053634643554688, 1.2437583208084106, 0.6672672629356384, 1.0824894905090332, -1.1405338048934937, -0.19218534231185913, 0.718498945236206, 0.6658342480659485, -1.1080944538116455, -0.051495544612407684, 1.0593206882476807, -0.10695692151784897, -0.8847600221633911, 1.0012890100479126, 0.13843423128128052, -0.949749231338501, -0.4549024701118469, 0.9434827566146851, -0.7743890881538391, -1.070935845375061, 2.260831117630005, 0.14314690232276917, -0.11130274832248688, 0.1544804871082306, 0.789095401763916, -1.1215088367462158, 0.6656084656715393, 1.3153774738311768, 0.5708997845649719, -0.3592052757740021, 0.3963399827480316, -2.4781150817871094, 0.6238403916358948, -0.6502695679664612, -0.23471881449222565, -1.0526484251022339, 0.5153329372406006, 0.9534730315208435, -0.5348511338233948, 1.4486687183380127, -2.1276028156280518, 0.5519852638244629, 0.2507448196411133, 0.22759462893009186, -0.7950518131256104, -0.04613832011818886, 1.470175862312317, 0.6920928359031677, 0.0020862456876784563, -0.587225079536438, 0.6228943467140198, -0.019611498340964317, -1.5767306089401245, -0.6233755350112915, -0.2905738651752472, -0.5944334268569946, 0.8138564825057983, 0.03720889613032341, -0.10481703281402588, -2.1818835735321045, -2.428076982498169, 1.5565972328186035, -1.3629670143127441, 0.30602115392684937, 0.20486892759799957], [-0.7208933234214783, 0.18528492748737335, 0.45541900396347046, -0.546308159828186, 0.4907134175300598, 0.28344041109085083, 0.05414154380559921, 0.4454735219478607, -0.6866012215614319, -0.0020109014585614204, 0.1873960942029953, 0.25812795758247375, 0.9168375134468079, 0.06656193733215332, 1.9219306707382202, -1.0100866556167603, -0.6936630010604858, 1.629640817642212, -0.7279607057571411, -0.7686179280281067, 0.7059991955757141, 1.0209912061691284, -0.12035232782363892, 0.4789889454841614, 0.41884833574295044, -0.05736347287893295, -0.5727914571762085, -1.8727738857269287, -0.36583662033081055, -0.27248480916023254, -0.24891553819179535, 1.1366904973983765, -1.0482293367385864, 0.04110381752252579, 0.1570010781288147, -0.9254932403564453, 0.8711993098258972, 1.4984537363052368, 1.311401605606079, 0.41670119762420654, 0.8583415150642395, -2.0013513565063477, -0.6461427807807922, 0.00589443976059556, -0.40256187319755554, -0.8454744815826416, -2.627201795578003, 1.9296289682388306, 0.0004023672954645008, -1.2162524461746216, 1.0618900060653687, -0.7672111988067627, 2.0451231002807617, -2.260556221008301, -0.19924211502075195, -1.3938945531845093, 0.211989164352417, 1.874069094657898, -0.8203643560409546, -0.0382269024848938, 1.2246893644332886, 0.018738990649580956, -0.012114285491406918, -0.4358404576778412, -0.010249640792608261, -0.7592236995697021, 0.7992653846740723, -0.5749174952507019, 0.3821181058883667, 0.2725287675857544, -0.02279457077383995, 0.2602168321609497, 0.8754417896270752, -0.016895566135644913, 1.3853181600570679, 1.5710787773132324, -0.8026222586631775, -1.0434341430664062, -0.3039321005344391, -0.46326944231987, 1.350358247756958, -1.3884649276733398, -0.8949071764945984, 0.6069228053092957, -0.6390014290809631, -0.345731258392334, -1.1723653078079224, -1.2651528120040894, 0.7159327864646912, -0.8617297410964966, -0.8614466786384583, 0.911225438117981, 0.2789554297924042, -0.14403244853019714, -1.1372623443603516, -1.3227328062057495, -0.19227297604084015, -1.5393507480621338, -0.14180900156497955, 1.2262628078460693, 0.2790403962135315, 0.2031538337469101, -1.0027111768722534, -0.14527487754821777, 0.30679839849472046, -1.432714819908142, 0.897739052772522, -1.0215762853622437, 0.03392666578292847, -0.6902816295623779, 1.8124850988388062, 0.6457597017288208, -1.5426859855651855, -1.3520127534866333, 1.6834603548049927, -0.9385566711425781, -0.8972668647766113, -0.20843468606472015, 2.0880210399627686, 0.6603980660438538, -0.006909775547683239, 0.6143787503242493, -0.18497015535831451, 0.8172205686569214, 0.8349901437759399, 1.1876310110092163, -1.3223320245742798, -0.6626423001289368], [0.30605727434158325, 0.7805796265602112, -0.5762644410133362, 0.9053855538368225, 0.6898064613342285, 1.1118930578231812, 0.9560966491699219, 0.023923195898532867, 0.749177873134613, 0.7723018527030945, -1.633603811264038, -1.4483963251113892, -1.5785495042800903, 0.5168479084968567, -0.4967825710773468, -0.8275672197341919, -1.023154854774475, 0.9673938155174255, -0.45184245705604553, 1.94573974609375, -0.40027713775634766, 0.57627934217453, 0.6822408437728882, 0.787309467792511, 1.2935668230056763, -0.8878408670425415, -0.6746721267700195, 0.2092379778623581, -0.12611617147922516, -0.7898651957511902, -1.272038221359253, -1.154111385345459, 0.35974758863449097, -0.3923361897468567, -0.7678265571594238, 0.9680098295211792, 0.46566537022590637, 1.3424370288848877, 0.12066590785980225, -1.0282782316207886, -0.23603320121765137, 0.4873284101486206, 0.28975096344947815, -1.3948094844818115, 0.9038524627685547, 0.44081029295921326, -0.7818917632102966, 1.4317923784255981, 0.1520344316959381, -2.038388729095459, -1.7443554401397705, -0.8890365362167358, 1.1599773168563843, -0.567842960357666, 0.9285789132118225, 1.4698901176452637, -1.766406536102295, 1.0223535299301147, -0.2965841293334961, -0.8975422978401184, 1.5368256568908691, 1.4003098011016846, -1.1807489395141602, -0.05319034680724144, 0.14308282732963562, 0.5500372052192688, -0.10911289602518082, -1.1609017848968506, -1.0586875677108765, -1.76366126537323, 0.08274933695793152, -2.3349416255950928, -0.249772846698761, -0.2318110167980194, 1.1279317140579224, 1.8277134895324707, -0.7305817604064941, 0.7079426050186157, -1.7616372108459473, -0.7438029646873474, 1.3686730861663818, -0.32507818937301636, -0.7709650993347168, -0.4703430235385895, 0.09025254845619202, 1.2789108753204346, 1.7081321477890015, 1.5486801862716675, 0.33983713388442993, 1.5340734720230103, 1.6609529256820679, -0.11761835217475891, 2.246685028076172, -0.07135418057441711, -0.8864744901657104, 2.2801976203918457, 1.5737059116363525, -2.137979507446289, 0.27830469608306885, 0.012455534189939499, 0.7386022210121155, -1.1112565994262695, -1.6606045961380005, -1.5095300674438477, 1.4867032766342163, 1.711819052696228, 0.5964983105659485, -0.8228355050086975, 0.5543556213378906, 0.2631390690803528, 0.21476666629314423, -0.788949191570282, -1.3118984699249268, -0.014826035127043724, 0.8112782835960388, 0.4681859314441681, -0.4941217601299286, -1.4772461652755737, -0.21608449518680573, -0.2947307527065277, 1.9123972654342651, 0.7903849482536316, -1.7483268976211548, -0.33621323108673096, -1.0296459197998047, 0.9140939712524414, -0.5667724013328552, -0.6238391995429993], [-0.28838443756103516, 1.229504942893982, 0.5171851515769958, -0.3425111770629883, -1.0916231870651245, -2.1754658222198486, 1.6473733186721802, 1.6628987789154053, -1.5112472772598267, -0.04371802881360054, 0.8869367837905884, 0.6296676397323608, -0.5113797783851624, -1.8524844646453857, -1.1645305156707764, 0.850124180316925, -0.18656465411186218, 0.7815761566162109, -1.5008487701416016, -0.1725071221590042, -0.5408879518508911, 0.17716485261917114, 0.11273666471242905, 0.4964173138141632, 0.07581649720668793, 2.0259571075439453, 0.758577823638916, 1.8644165992736816, -0.6116676330566406, 0.5548753142356873, 1.57390558719635, 0.6647818088531494, 0.8747321367263794, -0.3074638247489929, 0.05422951653599739, -0.9689679145812988, 1.008379340171814, 0.36940574645996094, 1.1206845045089722, -0.8768377900123596, -0.9008892774581909, 1.833605408668518, 0.44058194756507874, -0.15545055270195007, 1.2829148769378662, -1.1920866966247559, -0.5475942492485046, 0.23267191648483276, -0.35777634382247925, -0.3388253450393677, 0.849951982498169, 0.13151706755161285, 0.5241646766662598, 0.15361207723617554, 0.8555894494056702, 0.9088441133499146, -1.4814547300338745, -0.05494379624724388, 2.0448925495147705, -0.22995685040950775, 0.6095453500747681, -0.18966801464557648, -0.5160890817642212, 1.5451387166976929, 0.7977418899536133, -0.8996163606643677, -1.817826509475708, 0.7453457713127136, 0.9599490165710449, 0.5034705996513367, 0.2690267264842987, -0.17129702866077423, -0.7685145735740662, -0.8536059856414795, 0.251598984003067, -0.8873392343521118, -0.4415475130081177, -0.45370030403137207, 0.7965551018714905, 0.8141000270843506, -0.35147494077682495, -1.1994754076004028, -0.3949178457260132, -0.3215165436267853, 0.3266543447971344, 0.4249249994754791, -0.5484983325004578, -0.4020250141620636, 0.18373584747314453, -0.5228345394134521, 1.8765935897827148, -0.04458191990852356, 0.2814258933067322, -0.21456025540828705, -0.9685174226760864, -1.5531715154647827, -0.48738646507263184, 0.6101148724555969, 0.5405822992324829, 0.48090308904647827, 1.055571436882019, 0.24842219054698944, 1.1303478479385376, 0.736810564994812, -3.0928468704223633, -0.24174128472805023, -0.8659563064575195, -0.20934781432151794, -1.3702598810195923, 0.6585789322853088, 0.9507747888565063, -0.05988868325948715, -0.6564599871635437, 0.03135860338807106, 0.13738608360290527, -0.709484338760376, 0.2253796011209488, 0.8895920515060425, 0.29035812616348267, 0.18040165305137634, -0.5516225099563599, 0.08212706446647644, 2.2224576473236084, -0.3264574110507965, 0.7660802006721497, 0.029793277382850647, -0.8975294828414917, -2.3451290130615234], [-1.3499839305877686, 0.06062750890851021, -0.3219813108444214, 1.3383525609970093, 1.1430811882019043, -0.9785584807395935, -0.34183114767074585, 0.4132314622402191, 0.129116490483284, -2.026334524154663, 0.46033158898353577, 0.4593147039413452, 0.2684996426105499, 0.12683536112308502, -0.25417041778564453, -0.22995004057884216, 0.11163143068552017, 0.8777586817741394, 1.0489354133605957, -1.8943207263946533, 0.21465688943862915, -0.43817684054374695, 0.391426146030426, 0.6977729201316833, -0.9900363087654114, -0.13787828385829926, 0.9370670318603516, 1.5371960401535034, 0.6551438570022583, -0.07799139618873596, -0.09921342879533768, 0.26770853996276855, -0.5757303237915039, -0.4170158803462982, 0.23133555054664612, 0.6343455910682678, 0.14752477407455444, -0.04553627222776413, 0.5424863696098328, -1.2578184604644775, -0.12603537738323212, 0.25728946924209595, 0.9221214652061462, 0.3164994716644287, -1.1247317790985107, -1.39395010471344, -0.4134847819805145, 0.6580840349197388, -0.6690858006477356, -0.5816274285316467, 1.08963143825531, -0.9148760437965393, -0.15073904395103455, -0.22210177779197693, 0.2525902986526489, 0.0920630395412445, -1.2803846597671509, 0.4631401598453522, 0.08583683520555496, -1.1636006832122803, -0.6189832091331482, -1.2446705102920532, 1.6091760396957397, -0.7957372069358826, -0.5266205668449402, -0.2138223648071289, 0.48252204060554504, 0.9040921926498413, -0.6115872263908386, 0.04717395454645157, 0.10242775082588196, -0.011938015930354595, -0.17081256210803986, 0.7493060231208801, -2.9300637245178223, -0.5647062659263611, 1.5570248365402222, -2.3364417552948, -0.8715645670890808, -1.0353467464447021, -0.4096960127353668, 0.2835606336593628, 0.6829306483268738, 0.7026405334472656, 1.2410979270935059, -0.060086678713560104, 0.798767626285553, 1.1353378295898438, 1.1965785026550293, -0.16099271178245544, -2.0032665729522705, 0.2545342445373535, -0.95892333984375, 0.5541643500328064, -0.8521580100059509, 0.9454776048660278, 0.6307939887046814, -0.4652908742427826, -0.19315126538276672, 1.4110150337219238, 0.6856887340545654, -0.4185973107814789, -0.37123847007751465, -1.0416951179504395, -0.5673456788063049, -0.6250895261764526, 0.06242845952510834, 0.9493932723999023, 1.6285693645477295, 0.24276487529277802, -1.0353662967681885, -1.0187203884124756, 0.5531993508338928, 1.128753423690796, -0.29771628975868225, -0.7961167097091675, -0.744092583656311, -0.07826157659292221, 1.2638319730758667, -0.8122905492782593, -0.16019250452518463, -1.3095625638961792, 0.23010636866092682, -1.113036036491394, 0.5164093375205994, 1.135655403137207, 0.3679198622703552, -0.6410908102989197], [-0.3407280743122101, -0.3079412579536438, 0.3907443583011627, -0.6599884629249573, 1.2232311964035034, -0.16381549835205078, -0.45991188287734985, 0.0629974976181984, 0.8390544652938843, 1.1515673398971558, -0.46643972396850586, -0.3354640603065491, -0.5288702845573425, -0.6251317262649536, 0.9319895505905151, 0.8613195419311523, -2.0343258380889893, 0.79665607213974, -0.6527785062789917, 0.12571747601032257, -0.1814848631620407, -0.11063793301582336, 1.6140271425247192, 1.040432333946228, 1.257324457168579, -0.6302692294120789, -0.9156805872917175, 0.7687913179397583, 0.6032195687294006, -2.263765573501587, 0.6313089728355408, -1.3713654279708862, 0.07773379981517792, 0.6826238632202148, 0.1889321208000183, 0.79400634765625, 0.6795505881309509, -0.7338675260543823, -0.3766983449459076, 1.5377005338668823, 0.928898274898529, 0.7170214653015137, -1.4944186210632324, 0.71625155210495, 0.22482797503471375, 0.45376312732696533, 0.07134635746479034, -1.767893671989441, -1.260441780090332, -0.5003819465637207, 0.6094832420349121, -1.0835850238800049, -0.02360244281589985, 1.1314935684204102, 0.9463996291160583, 0.13995200395584106, 0.16100390255451202, -0.11303944885730743, 1.167875051498413, -0.43377912044525146, 0.234737828373909, -0.27475038170814514, -0.5139224529266357, 1.9010717868804932, -1.437422275543213, 0.11990046501159668, -0.8753716945648193, -1.2305419445037842, -1.3801349401474, 0.5373098850250244, -2.1335222721099854, 1.3358761072158813, 0.2721637785434723, -1.2694716453552246, 0.5529839992523193, 0.33918875455856323, 1.435856580734253, 1.4046992063522339, 1.175610899925232, 0.0844373032450676, 1.3487709760665894, 0.052460357546806335, -0.30999210476875305, -0.19201494753360748, 0.3621770143508911, 1.0983225107192993, 1.150497555732727, -1.6954927444458008, -2.2177507877349854, -0.9191922545433044, -0.009650071151554585, -0.6892582774162292, 1.5594483613967896, -0.6694242358207703, 1.5956991910934448, -2.0631513595581055, -1.0208760499954224, 0.3566441833972931, 0.7024881839752197, -0.34844449162483215, 0.9445003271102905, -1.2833163738250732, -0.6985456347465515, -0.10994455218315125, -0.7679070234298706, 1.486873984336853, -0.8883533477783203, 0.8756511211395264, -0.7162376642227173, 0.7560921311378479, 0.408598393201828, 0.6384705305099487, 0.17708081007003784, -2.5340497493743896, -0.7454557418823242, 0.7014496922492981, 1.1418957710266113, -0.08573832362890244, -0.3541591763496399, -1.1109603643417358, -1.439188003540039, -0.15973062813282013, -0.7309547662734985, -0.4110972583293915, -1.46150803565979, 0.1307549625635147, 0.5794295072555542, 0.6787751913070679], [-2.625915765762329, -1.1935524940490723, 0.5748487710952759, -0.3746599853038788, -0.675801157951355, -0.2498251497745514, -2.633260726928711, 1.4793974161148071, 0.7632115483283997, -1.3059399127960205, -0.14218296110630035, -0.8820458650588989, 0.36323440074920654, 0.4382186233997345, -0.6301828026771545, 0.8867761492729187, -1.4442570209503174, -0.5289490222930908, 0.5852635502815247, -1.8791794776916504, 0.4434504806995392, 1.0505753755569458, 0.7789092659950256, 0.16314220428466797, 2.7528536319732666, 0.5041229724884033, -0.3869071900844574, 0.5409283638000488, -0.592961311340332, -1.8853487968444824, -0.9773826599121094, -1.4412349462509155, 1.964673638343811, 0.7472817301750183, -0.28003254532814026, -0.1807965636253357, 2.1950571537017822, 0.560342013835907, -0.7517966032028198, -0.5301626920700073, -0.13342787325382233, -0.07718723267316818, 0.32631441950798035, 1.6074753999710083, -0.9367629885673523, -0.6767174601554871, 1.1965974569320679, 0.7920621037483215, -0.9186468124389648, 0.497546911239624, -0.915591299533844, -0.6218332648277283, -0.8286352753639221, 1.4714462757110596, 2.0959975719451904, -1.256091833114624, -0.9561044573783875, -2.4877371788024902, -0.4448241889476776, -0.8743978142738342, 0.691389799118042, 1.8945367336273193, 0.32484155893325806, 1.4972914457321167, 0.6437262296676636, 1.260390281677246, 2.350904703140259, -0.04809819534420967, 0.0812675952911377, 0.33761730790138245, 0.20024819672107697, 2.475346565246582, -0.6496871709823608, 0.7101728320121765, 1.653613805770874, 1.0444555282592773, -0.6848512887954712, 0.17924724519252777, 0.4814583957195282, -1.0006994009017944, 0.8309683799743652, -0.6392596960067749, -1.2369798421859741, -1.5519773960113525, 0.3708873987197876, -0.2809249758720398, -0.08213234692811966, 0.07023351639509201, -0.8921083211898804, 0.354800283908844, 0.052087508141994476, 2.347444772720337, 0.7537457346916199, -0.8622528910636902, -0.7624430060386658, 0.5667943954467773, 1.7403346300125122, 0.9797849059104919, 1.365064263343811, -0.9807199835777283, -0.4089040756225586, 0.3936479985713959, 1.6619925498962402, -0.6028218269348145, -0.10581842064857483, -0.8130231499671936, -0.7491234540939331, -0.07783469557762146, -0.849422037601471, -0.3052648901939392, -0.1756647229194641, -0.2105366736650467, 0.06418871879577637, 0.5225225687026978, 0.9621354937553406, 1.4337291717529297, -0.22147929668426514, 0.7500576376914978, -0.8710131645202637, 0.87315434217453, -0.49393653869628906, -0.5393940210342407, -1.6711318492889404, -0.849118173122406, 0.017742980271577835, -0.4696486294269562, 0.45195308327674866, -0.5333676934242249], [-0.07967420667409897, -2.109743595123291, -1.3277074098587036, -0.4604146480560303, 0.09054752439260483, -0.9151083827018738, -0.7620640397071838, 1.2757669687271118, -1.8830862045288086, -1.4414798021316528, 0.7427312731742859, 0.7059305310249329, -0.6562780737876892, 0.4392721951007843, 1.034710168838501, -1.0274097919464111, -0.5055595636367798, 0.5935952663421631, 1.5594464540481567, 0.06643616408109665, -0.9494897723197937, 0.05760333314538002, 0.47125664353370667, -0.12397126108407974, -1.3699417114257812, -0.31037136912345886, -0.9797860383987427, 1.0346486568450928, 1.4449267387390137, -0.4703214168548584, -0.5063058733940125, 0.811802327632904, -0.3540983498096466, 0.8964121341705322, 2.255685806274414, 0.15913026034832, -1.9806119203567505, 0.08407635986804962, -0.02730083093047142, -0.05144728720188141, 0.19079537689685822, 1.89110267162323, 0.4896867275238037, 0.40885549783706665, 0.3656867742538452, 0.36017847061157227, -1.1757837533950806, 0.3712160885334015, -0.37810447812080383, -1.4421483278274536, -0.4159139394760132, -1.8705990314483643, -0.051247674971818924, 0.5665778517723083, -0.5642237663269043, 2.897442579269409, 0.3586767911911011, 0.5347760915756226, 1.485940933227539, -0.3993053734302521, -0.581494152545929, -0.3031138479709625, 0.8932608366012573, 0.46228328347206116, -0.8110837936401367, 0.6672790050506592, -1.3478633165359497, 0.6067157983779907, 1.225077509880066, 0.41592589020729065, -0.5081217885017395, -0.5104361176490784, 0.845140814781189, 0.5770863890647888, -0.19935496151447296, 0.11900340765714645, -0.48481184244155884, -0.5088120102882385, -2.067775249481201, 0.44079023599624634, 0.2998671233654022, 0.2904575765132904, -0.7158374786376953, 0.26750659942626953, 1.3245660066604614, -1.2837944030761719, -0.8491933941841125, -0.8649922609329224, -0.07973187416791916, -0.9029890298843384, 0.6352540850639343, -0.22411026060581207, -0.21367652714252472, -0.03358088806271553, 0.26723888516426086, 0.9747563600540161, -0.8582379221916199, -1.6518343687057495, 1.5035018920898438, -0.2413250207901001, -0.1792101413011551, -0.2709765136241913, -0.07494820654392242, -0.9353859424591064, -1.1876858472824097, 0.7677134871482849, 0.07747901976108551, -0.3139905333518982, 1.2564581632614136, 1.1159981489181519, 0.5745071172714233, 0.0017274252604693174, -2.002943992614746, 0.8486572504043579, 1.3025579452514648, 0.7591692805290222, -2.756544828414917, -0.6997043490409851, 0.4729354977607727, 0.8161385655403137, -0.210682675242424, -1.0704617500305176, 1.245139241218567, 0.19773037731647491, -1.3668967485427856, -1.0795021057128906, 0.1745511144399643, -0.055112630128860474], [1.717882752418518, -0.9318499565124512, -0.9199734926223755, -0.07535926252603531, -0.2405010461807251, -0.13259191811084747, -0.37574484944343567, 0.3480541408061981, 0.40067803859710693, 0.06598590314388275, -0.2569146752357483, 0.20696714520454407, 1.3574628829956055, 0.8053176999092102, -1.7179279327392578, -0.49411699175834656, 0.550873875617981, 0.7724967002868652, 1.8653181791305542, -1.697320818901062, 2.579528570175171, 0.02493946999311447, -0.24030916392803192, 1.0044294595718384, 0.6725549697875977, -0.818178117275238, -0.14395993947982788, 0.710218071937561, -0.06518065184354782, -0.056428276002407074, 0.317402720451355, 0.9333640933036804, -0.5511026382446289, -0.8212307095527649, 0.6318318843841553, 0.4613930881023407, 0.38998693227767944, 0.46101441979408264, 0.9682820439338684, -0.6468054056167603, -0.8175780773162842, 0.7559724450111389, 0.8091515302658081, -0.15653133392333984, -1.8047385215759277, 0.3067604899406433, 0.8709015250205994, -1.009616732597351, 0.6547167897224426, -1.9760457277297974, 0.4021969139575958, 1.07175612449646, 1.9497607946395874, 1.8920643329620361, 0.2105308175086975, -0.26367637515068054, -0.8674807548522949, -0.38732394576072693, -0.5240328907966614, -1.574899435043335, -0.32203516364097595, 0.1175883412361145, -1.1957274675369263, 0.1065526232123375, 2.0370097160339355, 0.086405448615551, 1.2933838367462158, 1.6774152517318726, 0.9830180406570435, -0.027062885463237762, 0.7350164651870728, -1.0258151292800903, -1.5319827795028687, 1.7225896120071411, 1.092753291130066, -0.12908783555030823, -0.4239305853843689, 0.033127158880233765, 0.07020560652017593, -0.8427806496620178, 0.14255905151367188, -0.9563677906990051, 1.2131234407424927, 1.9115253686904907, 0.8352987170219421, -0.7886391878128052, 0.2979409694671631, -2.255265951156616, 1.018640160560608, 1.717935562133789, 0.007607111241668463, 0.28224825859069824, -1.3928309679031372, -1.2066535949707031, 0.7069695591926575, -0.912824273109436, 0.3481687307357788, -0.869526207447052, -1.4989831447601318, 0.5367974042892456, -0.19121825695037842, 0.16420194506645203, -2.6219539642333984, -1.2408348321914673, -0.19329112768173218, -1.5780730247497559, -1.2929788827896118, -0.5659037232398987, -0.008245224133133888, 2.280320167541504, 0.8324252367019653, 0.7103879451751709, -1.009933352470398, -1.1635541915893555, 0.3345074951648712, 0.6534022092819214, -0.8074853420257568, -1.0978403091430664, -0.7666744589805603, -0.2318626046180725, -0.09359872341156006, -0.809359610080719, -0.5176282525062561, 0.47109004855155945, 0.5794051885604858, 0.9284871220588684, 0.8512849807739258, 0.17897453904151917], [0.6994020938873291, 0.5937939882278442, -1.3514509201049805, -0.2973220646381378, -0.43040648102760315, -0.07940945774316788, 0.2189427614212036, 0.9215789437294006, 0.9453113675117493, 0.7006887197494507, -0.6620871424674988, -1.0314397811889648, 0.6808241605758667, -0.6787015199661255, 0.4804411232471466, -0.03414566069841385, -0.24848449230194092, 0.7231177091598511, 0.6766975522041321, 1.5451170206069946, -1.3542239665985107, 0.8850880861282349, -1.226577639579773, 0.7610995769500732, -0.14312970638275146, -0.14693476259708405, -0.06321121007204056, 0.01053378451615572, -0.2582240700721741, -1.8963673114776611, 0.7052164077758789, -0.7876380681991577, 0.42160528898239136, 1.9877680540084839, -1.094022512435913, -0.6303920149803162, 0.7687181830406189, -1.0768027305603027, -0.5082413554191589, -2.0456442832946777, -0.584266722202301, -1.2113103866577148, -0.8853459358215332, 0.6914160847663879, -1.1134300231933594, 0.04098283872008324, -0.37502622604370117, 1.2892727851867676, -0.5964002013206482, 0.04262232407927513, -0.47990357875823975, 0.2071327120065689, -1.0546557903289795, -0.6366490125656128, -0.048828545957803726, 0.4932273030281067, 0.9045617580413818, 0.47323936223983765, 3.4000980854034424, -0.45886486768722534, -0.04254930466413498, 0.535060465335846, 0.5208101272583008, 0.5251160264015198, 1.247256875038147, 1.2555363178253174, -1.2909544706344604, 0.5020138621330261, -1.1248146295547485, -0.5554494261741638, 1.1226774454116821, -0.8617632389068604, -0.8962042927742004, -0.7620439529418945, 0.6432371735572815, -0.7146930694580078, -0.737044632434845, 1.659058928489685, -0.9026188254356384, -0.45132550597190857, 1.5655964612960815, -1.7570319175720215, -0.9718447327613831, 1.3618324995040894, 0.41701361536979675, -2.605494976043701, -0.9109000563621521, 0.40472689270973206, -0.47540342807769775, 0.274722695350647, -0.570297122001648, -1.0885485410690308, 0.5644838809967041, -0.8018166422843933, 0.42988160252571106, 1.462821364402771, -2.4559478759765625, -2.2528505325317383, -0.17121505737304688, -0.6258945465087891, 1.9263641834259033, -0.5107753872871399, -0.6953617334365845, 0.5988368988037109, 0.06689313799142838, 1.299134612083435, 0.8709858655929565, -1.053917407989502, -1.4657683372497559, 0.44618409872055054, -0.93891841173172, 0.07642126083374023, 0.8090226054191589, 0.15264232456684113, 0.09031268209218979, 0.46702244877815247, -1.2430927753448486, -0.9188628196716309, -0.953346312046051, -1.6752721071243286, 0.35005244612693787, -0.33026590943336487, -0.16615664958953857, 0.8602778911590576, -2.7890403270721436, -0.8748360276222229, 1.3335163593292236, -0.8800442218780518], [-0.4469144642353058, 1.000727891921997, 0.08131477981805801, 0.05968043953180313, 2.3211898803710938, 0.06038729473948479, 1.7896249294281006, 1.115171194076538, 0.5857945084571838, 1.5714876651763916, -1.5191149711608887, 1.1752562522888184, -1.4523147344589233, -0.04677651450037956, -1.5163074731826782, 0.4986720383167267, 0.46383312344551086, 1.8473907709121704, 0.397869348526001, -0.6641848087310791, -1.6883164644241333, -0.8764596581459045, -1.6669001579284668, 1.6135187149047852, -0.02666744403541088, 1.726351261138916, 1.100170373916626, -0.11659368127584457, -0.20120380818843842, 0.4340260624885559, 0.1718139946460724, 1.4435471296310425, 0.18339036405086517, 1.0927722454071045, 1.8023244142532349, 1.4842793941497803, 1.4841892719268799, -1.6868553161621094, -0.18268385529518127, 0.2748543322086334, 1.5916060209274292, 0.9881958961486816, 0.034516479820013046, 0.3210408389568329, -0.6512587666511536, -0.670412003993988, -0.7896447777748108, -0.6672936677932739, 2.0057830810546875, 1.6097731590270996, 0.3425661623477936, 0.16030442714691162, -0.5131291151046753, 0.6804363131523132, -0.6458422541618347, -0.6593215465545654, -1.3470771312713623, 0.7412048578262329, -0.7820204496383667, 1.0243825912475586, -0.48364198207855225, -0.3604968786239624, 1.2991136312484741, 0.17136800289154053, 0.20314843952655792, 0.8004807829856873, -0.39217692613601685, 0.16086030006408691, 0.026522696018218994, 0.03652278333902359, -0.17103591561317444, -0.051535043865442276, 0.32150182127952576, -0.8651715517044067, -1.689283013343811, -0.8132241368293762, 0.7035687565803528, -0.6465286016464233, -1.9294100999832153, 0.12837818264961243, 1.2423149347305298, -2.0987389087677, -0.6112635135650635, 1.9383578300476074, -1.4755545854568481, -1.2421271800994873, 2.428337574005127, -0.09226848185062408, -1.4411873817443848, 0.17837317287921906, 0.9382153153419495, -1.5903048515319824, -0.05708297714591026, 1.787678599357605, 0.6746013760566711, -0.45495662093162537, 0.3750718832015991, 1.7381906509399414, -1.3245917558670044, 1.4414464235305786, 1.0225163698196411, 0.7969621419906616, 0.5724827647209167, 0.5353431701660156, -2.1764416694641113, -0.554124116897583, -0.6102178692817688, 1.089172124862671, 0.3025391697883606, 0.3867137134075165, 0.5238304138183594, 0.7948360443115234, 0.3084723651409149, 0.15559174120426178, 1.0974966287612915, 1.0299856662750244, 0.35855916142463684, 1.1325589418411255, 1.0137747526168823, -0.5106443166732788, -1.0077975988388062, -0.5712906122207642, 0.10489212721586227, -0.30565401911735535, -1.0956618785858154, 0.392406165599823, 1.188585877418518, 1.0984525680541992], [1.3615931272506714, 0.8669774532318115, 0.7991576790809631, -1.2112915515899658, -0.04705790802836418, -1.234076976776123, 1.6337519884109497, -0.4587683081626892, -0.9608466625213623, 0.7984347939491272, 0.7483689785003662, -0.5862395167350769, 0.5087453126907349, -0.3815731406211853, 2.0689737796783447, -1.6942059993743896, -0.9769577980041504, 1.0384624004364014, 0.8802613019943237, -0.1493622362613678, 0.9652931690216064, 0.028567718341946602, 1.6351697444915771, -0.3723636865615845, 2.4500985145568848, -0.4040619432926178, -0.7122130393981934, 0.7381603717803955, 0.057744018733501434, 0.5024459362030029, 0.7036677002906799, 0.9459052681922913, -1.6834429502487183, 0.3482443690299988, -0.9686744213104248, 0.7082986831665039, 0.028638839721679688, 0.26646900177001953, -0.0192844420671463, 1.0040550231933594, 0.5631778836250305, 0.0858723595738411, -0.054756924510002136, 1.2486916780471802, 3.002342462539673, -0.923733115196228, -0.3985023498535156, 1.4361165761947632, -0.5491176247596741, -0.6135327816009521, 0.4947872459888458, -0.9783979654312134, -0.852230966091156, 0.42476576566696167, -1.308567762374878, -1.32681405544281, 1.525182843208313, 0.19191129505634308, 0.6885896325111389, 0.14313474297523499, -0.9684855937957764, 0.33034858107566833, 1.6399650573730469, -0.783636748790741, -1.5315721035003662, -1.5093485116958618, -0.14814670383930206, 1.4978156089782715, 0.15128888189792633, 0.7141242623329163, -1.2200651168823242, -0.6885856986045837, -1.1594797372817993, -0.19136066734790802, 0.10765586048364639, -0.7311707139015198, 2.0676608085632324, -0.9363139271736145, 1.118293046951294, 1.7136963605880737, -1.1344449520111084, 2.041957139968872, 1.2475316524505615, -0.3996990919113159, 0.7678382396697998, -0.6794684529304504, 0.5157046318054199, -0.24603013694286346, 0.9859802722930908, 0.08339427411556244, -0.045368071645498276, 0.12339940667152405, 0.4561033546924591, -0.20432767271995544, -2.699751138687134, 0.8260254263877869, -0.35508888959884644, 0.6805624961853027, 0.41274720430374146, 0.7492559552192688, -0.6114166975021362, 1.005536675453186, 0.47609710693359375, 1.0466947555541992, 0.5204699039459229, 0.47267207503318787, -0.7015483379364014, 0.16302815079689026, -3.031088352203369, 1.3788878917694092, -0.3671608567237854, 1.8174378871917725, -0.1042625829577446, 1.0941245555877686, -0.5944243669509888, -1.1985504627227783, -0.3623485565185547, 1.6425305604934692, -0.9025560617446899, -2.265734910964966, 1.7907030582427979, 0.6264730095863342, 0.31168317794799805, 0.3415835499763489, -1.6316750049591064, -0.6966519951820374, -0.46264374256134033, 0.8113325834274292], [-0.7616905570030212, -0.1472635269165039, -1.9798429012298584, 0.9278824925422668, -1.4816519021987915, 1.6365540027618408, -0.16957029700279236, 0.23869021236896515, 1.5750863552093506, 0.9888615608215332, -1.679919719696045, -0.0199175663292408, 1.433064579963684, -0.4184965491294861, -0.7837279438972473, -0.20794853568077087, -0.8562656044960022, -0.7659407258033752, 0.22080321609973907, 0.637130856513977, -1.193640112876892, 0.591628909111023, 0.9109915494918823, 0.09897714853286743, 0.6827710270881653, 0.8102020621299744, -0.6294366717338562, -0.03428465500473976, -0.1754053682088852, 0.2912119925022125, -0.17252607643604279, -0.594326376914978, -1.4347079992294312, -0.008658280596137047, -1.7570245265960693, -0.4098082482814789, -0.12471843510866165, -0.6132400631904602, -0.35746005177497864, 0.5472580194473267, 1.4728163480758667, 0.4515949785709381, -0.9966200590133667, 1.1708340644836426, 0.3919357359409332, -0.3648669123649597, -0.4394185245037079, -0.1360563039779663, 0.16925457119941711, 0.8205835223197937, 1.128564715385437, -1.255501627922058, 0.6922328472137451, -0.30974704027175903, -1.5091509819030762, 0.5397696495056152, -1.0769466161727905, -0.020918667316436768, 1.2552061080932617, 1.8878586292266846, 0.25979137420654297, 0.12304309755563736, -0.14735841751098633, -0.6023425459861755, -0.3140285611152649, 0.2938038110733032, 0.24365274608135223, -0.11879279464483261, -0.2166012078523636, -0.24259334802627563, 0.28161314129829407, 1.1917800903320312, -1.7205853462219238, -0.8792905807495117, -0.6018775701522827, -0.45633038878440857, 1.6391876935958862, -0.42586061358451843, -0.2828163206577301, 3.129873752593994, -1.0636619329452515, -0.19620586931705475, -0.5760197043418884, -1.3199502229690552, -0.6254155039787292, 0.5968660116195679, 1.2479255199432373, 0.15276218950748444, 0.8378325700759888, -0.5135993361473083, 0.1514149010181427, -0.4068385660648346, -1.379028081893921, 0.18777120113372803, 1.3630552291870117, 0.45070621371269226, 0.4265548288822174, 1.1731681823730469, 0.5286651253700256, -0.035651665180921555, 0.946568489074707, 1.7727556228637695, 1.5304913520812988, 0.11873123794794083, -2.0215187072753906, 0.8345410823822021, -0.8745594620704651, -0.6946808695793152, -0.7623348832130432, 1.2036094665527344, -0.2351565957069397, -0.8544948101043701, 1.9033949375152588, 0.0354611873626709, -0.7574124336242676, -0.8532359004020691, 0.518759548664093, -0.6250362992286682, 0.7991009950637817, 0.42186808586120605, -1.2880386114120483, -0.04323114454746246, 0.01030031405389309, -1.9575167894363403, 0.2993778884410858, 1.9470949172973633, -1.4301625490188599, 1.429895281791687], [-0.40304678678512573, 0.04719771072268486, 0.44587090611457825, 2.323974847793579, 0.3294289708137512, 0.8277478218078613, -0.05852990597486496, 1.1226316690444946, 0.40380772948265076, -0.7932400107383728, -0.46170827746391296, 1.7773520946502686, -0.025660935789346695, 0.21764835715293884, 0.7126398682594299, -0.04630277678370476, 1.3923391103744507, 0.39195436239242554, -0.6036860346794128, -0.01952611282467842, 0.6470229029655457, 0.18000881373882294, -1.3446656465530396, -2.0622951984405518, 1.4366295337677002, 1.238539695739746, 1.3421047925949097, 2.1538209915161133, -0.25371241569519043, -0.6666059494018555, 0.16450218856334686, -0.0960908830165863, 1.804192304611206, 0.3329637944698334, 0.5147827863693237, 0.37653210759162903, 0.472869873046875, -0.5151407718658447, -0.5209505558013916, 0.05387145280838013, -0.027817782014608383, 0.33361876010894775, -0.40234503149986267, 0.2387198954820633, -0.5030383467674255, -1.8359342813491821, -0.36076903343200684, -0.08087658882141113, 1.3900736570358276, -0.4313397705554962, -1.6098555326461792, -1.5007500648498535, 0.34332212805747986, -0.7462628483772278, -0.8320144414901733, -0.5163279175758362, -0.30625420808792114, -1.1556899547576904, -1.663832664489746, 0.3223992884159088, 0.06830602884292603, -0.8244823813438416, 0.563687801361084, -0.6722869873046875, -0.752914547920227, 0.4358616769313812, -0.059536851942539215, 0.32377883791923523, -0.46079742908477783, 1.1906940937042236, -0.190063938498497, 0.03166862577199936, -0.9272366166114807, 0.03995983302593231, -1.2361425161361694, 0.3322316110134125, 0.666290819644928, 0.705091118812561, 1.6006020307540894, 1.2859079837799072, -1.5531690120697021, -1.0251591205596924, 0.10325460880994797, -0.5563613772392273, -1.3508397340774536, 0.32489246129989624, -1.437947154045105, 0.1371431201696396, -0.07744815945625305, 0.6778496503829956, -0.4774189889431, 0.21884270012378693, -0.8007838726043701, -1.6546850204467773, -0.1130615845322609, 0.04459838941693306, -0.4444480240345001, -0.05256856977939606, 1.436376929283142, -0.5273035764694214, 1.387003779411316, 0.4233933389186859, 0.9594221115112305, 1.9103609323501587, 0.5725116729736328, 2.6175646781921387, 0.09397182613611221, 1.4113237857818604, 1.2778173685073853, 1.7331676483154297, -1.285401463508606, -1.726671576499939, 0.9870120882987976, -0.6104874610900879, 0.12094791978597641, 0.43965020775794983, -1.1330231428146362, 0.4147966504096985, 0.8183626532554626, -0.5138436555862427, 0.7894700765609741, -0.7983747720718384, 0.35713833570480347, 1.7087634801864624, 1.179539680480957, -1.0881404876708984, -1.516024112701416, -0.3078314960002899], [0.7955756783485413, -0.024998711422085762, -1.2809679508209229, 0.835421621799469, 0.9936002492904663, -1.0342066287994385, -0.774443507194519, -0.6773445010185242, 0.8375656604766846, 1.518025517463684, -1.8602293729782104, -1.4363007545471191, -0.7984493970870972, 0.29711318016052246, -0.3672723174095154, 0.9406287670135498, 0.43116235733032227, -0.6113520860671997, 0.7184137105941772, -0.17478154599666595, 0.9335442185401917, -1.000120759010315, -0.5918848514556885, -0.04476618766784668, -0.3825332820415497, -0.9412626028060913, 1.7847212553024292, 0.6774380207061768, -1.86476731300354, -0.2517193853855133, 1.1960034370422363, -0.6493229866027832, -0.18657642602920532, -0.2672053575515747, -0.6680480241775513, -0.45109865069389343, -0.6825746893882751, 1.6258264780044556, -0.25636985898017883, -0.26357680559158325, -0.2963077127933502, -1.286354899406433, 0.9961406588554382, 0.34482046961784363, 2.2250304222106934, 0.5171441435813904, 0.35630422830581665, -2.578641891479492, 0.6089588403701782, -1.5352036952972412, -0.6833320260047913, -2.1156089305877686, -1.0137381553649902, -0.4647615849971771, -0.14631786942481995, -1.5687357187271118, 0.8240407705307007, 0.26248374581336975, -1.2144736051559448, 1.348787546157837, -0.9157403707504272, -1.0890779495239258, -1.3164538145065308, -0.9862922430038452, 0.7228510975837708, 0.7568955421447754, 0.06230717897415161, -0.6567539572715759, 0.1311325877904892, -0.15687133371829987, -0.1338246613740921, -1.1566603183746338, -0.471213698387146, 2.2463603019714355, 0.17334023118019104, 1.2733060121536255, -1.8950594663619995, -0.03881462663412094, 1.1324670314788818, 0.2172582894563675, 0.35165631771087646, -0.6164163947105408, -0.6208509206771851, 0.011602486483752728, -1.3361016511917114, -0.28197920322418213, 1.279272437095642, -0.24092425405979156, 0.06589486449956894, -0.526326060295105, 2.247493267059326, -1.0580452680587769, -1.412348985671997, -0.620581328868866, 1.8499677181243896, 1.383234977722168, -0.8550403714179993, -2.484227418899536, 0.7802193760871887, -0.4661449193954468, -1.6515763998031616, 1.017671823501587, -0.68281489610672, 0.7022343277931213, 0.13473553955554962, -0.7019177675247192, 0.5266043543815613, -0.7800302505493164, 0.48479151725769043, 1.1473554372787476, -0.6842763423919678, 0.18890568614006042, 2.218477725982666, 1.5028871297836304, -0.9683089852333069, -1.0823248624801636, 1.205339789390564, 0.6695118546485901, -1.0212308168411255, -0.7980400323867798, 0.2692992687225342, -0.30594971776008606, 0.7694517970085144, 0.04423872381448746, 0.025275234133005142, 0.1589653193950653, 1.6828206777572632, 0.8220945000648499]], [[0.2009749412536621, -0.2522408962249756, 2.846161365509033, 0.7693058252334595, -0.42246049642562866, -1.1862107515335083, 1.0818560123443604, 0.5657727122306824, 0.4896753132343292, 0.5148794651031494, -0.35115617513656616, 1.4170728921890259, -0.5747695565223694, 2.0288944244384766, 0.6464827060699463, 0.7480242848396301, 1.515183448791504, 1.2418891191482544, -1.1880265474319458, 2.6015625, -0.7523059248924255, 0.9635169506072998, -0.34333401918411255, 2.0430469512939453, -0.6607273817062378, 0.17714114487171173, 0.3797229528427124, -0.23435869812965393, -0.3135731518268585, 0.3647592067718506, -0.9422955513000488, 1.5605285167694092, 1.0989971160888672, -1.1866880655288696, 0.33838751912117004, 0.5668931603431702, 1.7071924209594727, 1.050128698348999, 1.0623433589935303, 0.15776029229164124, -0.34356218576431274, -1.8990380764007568, -1.0690668821334839, -0.12711018323898315, 1.0068399906158447, 0.30502283573150635, 1.6751421689987183, 0.11353346705436707, -0.3752429783344269, 0.3477734625339508, 0.08108556270599365, -0.40192827582359314, 1.5125751495361328, 0.24439682066440582, -1.7143176794052124, -0.8776612877845764, -0.09676235169172287, 1.0145925283432007, -0.3479854166507721, 1.190712332725525, 2.3226842880249023, -0.30526840686798096, 0.37486305832862854, -0.31853508949279785, 0.35942310094833374, 0.5502527356147766, 1.332313060760498, -0.6194548010826111, -0.5930128693580627, 0.6802127957344055, 0.1960754543542862, -0.7187647223472595, -0.6969466805458069, 1.7146047353744507, -1.0400179624557495, -0.7949329614639282, -0.038553230464458466, 1.1099501848220825, -1.0766558647155762, 1.7626996040344238, 1.6422107219696045, -1.306127667427063, 0.8031930923461914, 0.21470589935779572, -1.1133352518081665, -1.1548858880996704, 0.18403957784175873, 0.9270617365837097, 0.17460399866104126, 0.48664188385009766, 1.226959466934204, -1.4970895051956177, 1.0633573532104492, 0.8274205923080444, 2.1267073154449463, 0.39388811588287354, -0.5062462091445923, 0.2959040403366089, -0.6502147912979126, -1.7308423519134521, 0.11111972481012344, -2.0101122856140137, -1.4843487739562988, 1.0196402072906494, -2.0129940509796143, 1.2303463220596313, -1.7225888967514038, -0.3906906843185425, 1.7793406248092651, 0.1820356398820877, -1.2738769054412842, 0.7689806222915649, 0.6053522229194641, -1.7322156429290771, 1.0134453773498535, 0.7932992577552795, 0.8956733345985413, 0.26381370425224304, 0.07067001610994339, 2.791790246963501, -0.9509374499320984, 1.627804160118103, 0.45422595739364624, 0.30514466762542725, -0.5196285843849182, 1.3068486452102661, -1.4362330436706543, -0.4582448899745941], [0.7173287868499756, -0.21838490664958954, 0.4527679681777954, 1.178863286972046, 1.8468784093856812, -0.6322828531265259, -0.18818515539169312, 1.7455631494522095, 0.09165128320455551, -0.6396299004554749, 1.4920521974563599, -0.04057512804865837, -0.5294369459152222, 2.4980337619781494, -0.3622428774833679, 2.599586248397827, -0.42273348569869995, -0.59418123960495, -0.8070164322853088, 0.9112259745597839, -0.5739063620567322, 0.6924768686294556, 0.45251893997192383, -0.20801489055156708, -1.1109821796417236, 0.5589223504066467, -0.7979239821434021, -1.1907012462615967, 1.0319103002548218, -0.24923774600028992, 0.14593425393104553, 0.0653180181980133, -2.2027859687805176, 0.2734416425228119, 0.38130417466163635, -0.006107541732490063, -0.9867634177207947, -0.6501626372337341, -2.886587619781494, 0.4775576889514923, -0.41177886724472046, 1.1364424228668213, -1.410762906074524, 1.08281672000885, -0.6485987305641174, -0.20122447609901428, 0.9837484359741211, 0.2992566227912903, 0.8136546015739441, 0.08031371980905533, -1.1939681768417358, 0.6339618563652039, 0.8154175281524658, 0.592093825340271, 0.9512891173362732, -0.37594330310821533, -0.6029202938079834, -0.3748472034931183, 0.8526932001113892, -1.134995460510254, -0.7145019769668579, -0.6596623063087463, 1.3807947635650635, 0.9011701345443726, 0.9314730167388916, 0.44778090715408325, -0.46030569076538086, 0.44320330023765564, 0.5135899186134338, -0.8367509841918945, 1.923082947731018, -0.41668158769607544, -1.6713978052139282, -0.28788015246391296, -0.02860921621322632, -1.1101305484771729, -0.5480583310127258, 1.7321521043777466, -0.7010091543197632, 0.9912029504776001, 0.282235324382782, -2.074864625930786, 0.4869248867034912, 0.7744019031524658, 0.8179914951324463, 2.48795747756958, -1.0634931325912476, 0.7967175841331482, 0.17433132231235504, -0.4458339512348175, 0.7263875603675842, -0.3324293792247772, -0.07848919183015823, -0.17295867204666138, 1.9001023769378662, 0.7012487649917603, -0.2231179177761078, 1.561545729637146, -1.4367656707763672, -0.20046298205852509, -0.09231311827898026, 0.08488225936889648, 0.25516897439956665, -1.8285596370697021, 0.1620519608259201, -0.5566874146461487, 2.277808427810669, -0.018159521743655205, 0.6638153195381165, 1.5135010480880737, -2.045621156692505, 0.31181100010871887, 0.154279425740242, -1.6535488367080688, 0.07081645727157593, -0.5628131031990051, 0.02509542740881443, 0.40999555587768555, 2.214585304260254, 0.6049882769584656, -0.005475587677210569, -1.019637942314148, -0.06562685966491699, -0.302105575799942, 0.7837657928466797, -0.6706581115722656, -0.2161470502614975, -0.280788391828537], [-0.38678523898124695, -0.9022005796432495, -1.3982551097869873, 1.0452357530593872, -0.4263770580291748, -0.22036689519882202, -1.711543083190918, 0.0037273711059242487, -0.44860154390335083, -1.7007102966308594, -0.19354330003261566, 0.03685760498046875, 3.079911470413208, 0.552911639213562, 1.4051834344863892, 0.034019652754068375, -2.5717833042144775, 1.2073469161987305, 0.703353762626648, 0.23136745393276215, -0.45486631989479065, -0.05304686352610588, 0.4601863622665405, -0.03222004696726799, -0.2613486051559448, 0.5350033640861511, 0.04246513918042183, -0.5834996104240417, 0.3578767478466034, -0.9448713064193726, -0.9362075328826904, 2.083305597305298, 1.168973445892334, -1.1893702745437622, 0.8531519174575806, -0.22227831184864044, -0.6611782908439636, 1.557335615158081, -1.1757346391677856, -1.5227755308151245, -0.6039319634437561, 1.6499003171920776, -2.370959997177124, -0.2868214547634125, 0.7021716237068176, 0.7164183259010315, -0.8168716430664062, 0.35939502716064453, 0.47850486636161804, 0.09313654899597168, 0.14880870282649994, -1.3321733474731445, 0.2451324760913849, 1.1490886211395264, 0.16858035326004028, -0.04937963932752609, -0.5562028288841248, -0.921130359172821, -0.49899473786354065, 0.6432260870933533, 0.7399080991744995, 1.5344384908676147, -0.09568493068218231, 0.7624797224998474, -0.2297121286392212, -0.3894725739955902, -1.6271101236343384, -0.2703935503959656, 0.023147927597165108, -1.975286841392517, 1.4794082641601562, 0.05860164761543274, -1.1143244504928589, 1.329796314239502, 0.06266528367996216, 1.3144217729568481, 0.8833491206169128, -0.20920473337173462, -0.41359275579452515, 1.072675347328186, 2.6069226264953613, -1.0139013528823853, -0.33493924140930176, -0.1472742110490799, -0.34583038091659546, 1.3963232040405273, -2.000715732574463, 0.1637258231639862, -0.809829592704773, 0.7525348663330078, 0.10557477176189423, 0.7747476100921631, 0.5297678709030151, 1.6789084672927856, 0.7363102436065674, 0.9056556224822998, 0.6275277733802795, -1.1255184412002563, -1.1042040586471558, -0.3718646764755249, -0.9044852256774902, -1.4091970920562744, -0.48704856634140015, 1.0578413009643555, 1.3486013412475586, 0.9902215600013733, 0.6943559646606445, 1.6642463207244873, -2.203881025314331, 1.7155767679214478, 1.3213833570480347, -1.0399991273880005, -0.6862095594406128, -1.0939253568649292, 1.7572849988937378, 1.3900169134140015, -0.33386579155921936, 0.3106643259525299, -0.5356333255767822, -0.010475284419953823, 0.2650395333766937, -0.36901015043258667, 0.6537225842475891, 0.2568357586860657, 0.3068596422672272, 0.08357683569192886, -0.6349881887435913, -0.4669756293296814], [0.30218473076820374, 0.29625532031059265, 0.07462838292121887, 0.5482298731803894, 2.313363790512085, -0.5316030383110046, 1.2037460803985596, 1.18982994556427, -1.0135120153427124, -0.7125081419944763, -0.7263976335525513, -0.9566313028335571, -2.5487139225006104, -0.376609206199646, -0.3789045214653015, 1.5112712383270264, -0.8816004991531372, 0.7443808317184448, 0.4142199456691742, -0.4549448788166046, -1.4116460084915161, -0.3323008418083191, 0.28023621439933777, -0.48606669902801514, -0.5914356112480164, 1.516420841217041, -0.2953571081161499, -0.09342820942401886, -0.03323783352971077, -0.29095473885536194, -0.12538373470306396, -0.44820669293403625, 1.0413798093795776, -0.30158594250679016, 0.08167089521884918, -0.10116071999073029, -1.9555954933166504, -1.0864349603652954, -0.4109610617160797, -1.863466501235962, -0.49245941638946533, -0.048911139369010925, -0.392719030380249, 1.068839430809021, 0.711613118648529, -1.0142844915390015, -0.8631510734558105, -1.0972304344177246, -0.4432094395160675, 0.1819646656513214, 0.141837477684021, -0.8855059146881104, -0.034481827169656754, -0.7932425737380981, 0.6870622038841248, 0.4353071451187134, -1.1944787502288818, -0.07163519412279129, 0.4419425427913666, 0.014060345478355885, 0.3731509745121002, 0.1762424260377884, -0.9643986821174622, 1.0028254985809326, 0.6005971431732178, -0.44708675146102905, 0.8145526647567749, -0.11957433819770813, -1.7837821245193481, 1.1259880065917969, -0.8637527227401733, -1.3464183807373047, 2.1443464756011963, 1.3992756605148315, 0.08617573976516724, -0.8485146760940552, 1.3685312271118164, -2.461378812789917, 1.316941738128662, -1.2866848707199097, -1.491437554359436, 0.9718702435493469, 0.0772930458188057, -0.2114088386297226, 0.36380672454833984, 1.1756490468978882, -0.9564427137374878, -0.5981624722480774, 1.7362021207809448, 0.06377802789211273, 0.44193047285079956, -1.7097113132476807, 1.6084879636764526, -0.6434139013290405, 0.7879130244255066, -0.021630430594086647, -1.2149471044540405, 0.7591109275817871, 2.6008658409118652, 0.2892735004425049, 1.809740424156189, -0.24910277128219604, 0.30733755230903625, 1.0540441274642944, 0.6171450614929199, 1.1401926279067993, 0.870678722858429, -1.1602407693862915, -0.17436684668064117, 0.4968416392803192, 0.32917144894599915, -0.6492136716842651, -3.28670072555542, -0.8792588710784912, -2.188832998275757, -0.40977299213409424, -0.2676626443862915, 1.1904877424240112, -0.42376238107681274, 0.2797427177429199, -0.0412435382604599, 0.691697895526886, 1.0288993120193481, 0.6325109004974365, 1.4895764589309692, 0.0047184014692902565, -0.2838986814022064, -0.11419818550348282], [-0.8861337304115295, -0.7380744814872742, 0.4665655493736267, 0.06772483140230179, 1.7719248533248901, 0.1928306519985199, 1.2601430416107178, -0.33082669973373413, 0.3332009017467499, 1.1279289722442627, -0.10607048869132996, -0.6711505055427551, -0.19467923045158386, 1.501279354095459, 1.1448591947555542, -0.5066116452217102, -0.7578549385070801, 0.02179073914885521, 2.009243965148926, -0.13032890856266022, -0.16092807054519653, 2.3670401573181152, -1.2170931100845337, -0.6821672320365906, -0.34282612800598145, -0.9445793032646179, 1.3376498222351074, 1.235863447189331, 0.08076786994934082, -1.7663257122039795, -0.5160020589828491, -0.7881813049316406, -0.6789391040802002, 0.1041254997253418, -0.26169899106025696, -0.29844915866851807, 0.6081106066703796, 2.5140268802642822, 1.6707572937011719, -1.5572458505630493, -1.8602792024612427, -0.04404795542359352, -1.451301097869873, 0.6373413801193237, -0.20589186251163483, 0.31489190459251404, -0.5459263920783997, -0.5752114057540894, 0.14485910534858704, -0.11404475569725037, -0.4989098310470581, 1.221818208694458, -0.565183162689209, -1.4875138998031616, 0.31862080097198486, -0.5962597727775574, -0.4784686863422394, -0.3911251425743103, 0.14637954533100128, 0.3203046917915344, 1.6255974769592285, 1.844171404838562, 0.7632209062576294, -0.03942939266562462, -1.023679494857788, -0.3694736361503601, -0.222879558801651, 0.04306056722998619, 0.16070976853370667, -0.12683598697185516, -0.3476521074771881, 0.1904323250055313, 1.9037792682647705, 0.8630585074424744, -0.6098434925079346, -0.8529801368713379, -0.37253931164741516, -1.628836750984192, -0.13163205981254578, 0.4669707715511322, -0.4390566349029541, -1.0264604091644287, -0.4839082658290863, -0.802036464214325, -0.6827445030212402, -0.8760815858840942, -0.20208565890789032, -0.13325200974941254, 0.4618125557899475, -0.8143029808998108, -1.277058482170105, -0.1654796302318573, -0.6907446980476379, -0.3499981760978699, -0.287627637386322, 2.048851490020752, 0.6199853420257568, -0.6461458802223206, 0.760363757610321, 1.07809317111969, -1.238080382347107, -0.6610357761383057, -1.6711885929107666, -1.3368643522262573, -0.6008007526397705, 0.4178794324398041, 0.30344003438949585, -1.749870777130127, -0.4880713224411011, -0.6713058352470398, -0.630721390247345, 0.7499876022338867, 2.05959153175354, 0.7630696296691895, -0.9409856796264648, -1.9316719770431519, 0.21837569773197174, 0.9242151379585266, -1.194369912147522, -1.4027975797653198, -0.018851125612854958, 0.3138522207736969, 1.1567635536193848, -0.23008699715137482, -0.010783514007925987, 0.6261131167411804, -1.3918709754943848, -0.8188285231590271], [0.11760982871055603, 0.03490365296602249, -0.9927197098731995, 1.4180313348770142, 0.368850976228714, 0.015595141798257828, 0.0060517443343997, 1.119081974029541, 2.7336184978485107, 0.9431185722351074, 0.45732995867729187, 0.08602134138345718, 1.6188925504684448, 1.3270655870437622, 1.0700968503952026, -0.7362717986106873, -1.1965380907058716, -0.17400862276554108, 0.9125357270240784, -1.3047950267791748, -1.3123716115951538, -0.5795783996582031, 0.6095479130744934, 0.13652518391609192, -1.0166906118392944, 1.0229226350784302, 0.16805584728717804, -0.8471091389656067, -1.307631492614746, -0.8235839605331421, -0.10373801738023758, 0.9897238612174988, -0.404402494430542, -0.5395655632019043, -1.0888854265213013, -0.4311921298503876, 0.47539159655570984, 1.0465842485427856, -0.18413259088993073, -1.086097240447998, -0.02582717500627041, -1.5836678743362427, 1.2003164291381836, -0.7750840187072754, 1.107026219367981, 0.6267087459564209, -1.6583824157714844, -1.4748811721801758, 2.1471917629241943, 0.3815787732601166, 0.3438578248023987, -0.19730065762996674, -0.053307678550481796, 1.657320261001587, -0.08010166138410568, -2.364060401916504, -2.0126566886901855, -0.17154182493686676, -1.799431562423706, -0.7445488572120667, -0.9245691895484924, 0.5561830997467041, -0.2871444523334503, 1.6467735767364502, 1.7457201480865479, -0.49704673886299133, -0.09696535766124725, 1.2725026607513428, 0.21161724627017975, -0.4498274624347687, -0.9357134103775024, -1.4135496616363525, -0.4825875163078308, -0.8979753255844116, -0.9298153519630432, 0.23331354558467865, 0.4871895909309387, -0.6875078678131104, -1.6678645610809326, 1.7604973316192627, -1.4558889865875244, 0.43982774019241333, -0.19211821258068085, -0.448419451713562, -0.5191372036933899, -1.6046706438064575, -0.4966089427471161, 0.8770402669906616, -0.126015305519104, 0.703163206577301, -1.8034168481826782, -0.7615686655044556, 0.5467992424964905, -2.0590953826904297, -0.2911040782928467, 2.69280743598938, 0.2947634160518646, -0.4698368012905121, -0.023007279261946678, -1.4757975339889526, 0.7281574010848999, -0.03640224412083626, 1.253970742225647, -0.5324147343635559, 0.9078226089477539, -0.5964961051940918, 1.6121801137924194, -0.4822220802307129, -1.5799274444580078, 0.36025911569595337, 0.6750175356864929, -0.05411510542035103, -0.7647708654403687, -3.011286497116089, 0.4064580202102661, 0.3226432800292969, -1.0811043977737427, 0.15214231610298157, -1.4682763814926147, -1.2066470384597778, -0.9456303715705872, -0.559596598148346, 1.0153634548187256, -2.5075345039367676, -1.7798893451690674, 0.08910446614027023, -3.0161406993865967, -0.9064774513244629], [1.1530022621154785, -1.0543205738067627, -2.0053839683532715, -1.2453234195709229, 0.06617392599582672, -1.0267608165740967, -0.9956804513931274, 0.36519354581832886, -1.2915030717849731, 0.3961809575557709, 0.22800995409488678, -1.0990089178085327, -0.7736694812774658, 0.06460294127464294, -0.9988568425178528, 0.36273959279060364, 2.0043833255767822, 0.16123923659324646, 0.6248683929443359, -1.6791263818740845, 0.36442551016807556, 0.14377963542938232, -2.3711495399475098, -0.36200055480003357, -1.147143840789795, -0.8502717614173889, 0.3805256187915802, -0.5354295372962952, 1.8703656196594238, -0.06526127457618713, -0.12044999748468399, 0.68498295545578, 1.0170186758041382, 0.5524775981903076, 0.6103832125663757, -0.7707155346870422, 0.8666180968284607, 1.9276334047317505, -0.18743710219860077, 0.5562566518783569, -0.31391844153404236, 0.21393872797489166, 0.7242476940155029, -0.2037748545408249, -1.124663233757019, -1.4231692552566528, -0.06404270976781845, -1.1961958408355713, 0.5774050354957581, -0.6883378028869629, -0.9239925742149353, -0.6395521759986877, 1.6854751110076904, -0.38805273175239563, -0.4345860481262207, 0.9339340329170227, -0.7414964437484741, 1.419068694114685, 0.11248072981834412, 0.8174273371696472, -0.5197839140892029, -0.7759788036346436, 1.7725625038146973, -1.4730749130249023, 0.6340996623039246, -0.1281394213438034, -2.3907217979431152, -2.1720051765441895, 0.6290239095687866, 1.7836710214614868, -0.8944361805915833, 0.9563594460487366, -0.65059494972229, 0.016416195780038834, 1.0234795808792114, 1.8637253046035767, 1.0414931774139404, -0.05101141333580017, 0.8719049096107483, -1.0751186609268188, 0.7980940937995911, -0.07888739556074142, 0.21125271916389465, 1.6480457782745361, -0.033492837101221085, 1.147849440574646, -0.6570510864257812, -0.45919492840766907, 0.35342490673065186, -0.17796999216079712, 0.6729287505149841, 0.03562609478831291, -0.5551798343658447, 0.7902113199234009, 1.6121551990509033, -2.9632925987243652, -0.2732638418674469, 0.5194926857948303, 1.0779318809509277, -0.19225935637950897, -0.9159303307533264, -1.6836005449295044, 0.14041078090667725, -0.9114311337471008, 1.3384966850280762, -0.603080153465271, 1.1728111505508423, -0.10308866202831268, 1.6388627290725708, -0.44753625988960266, 1.204994797706604, 0.7133291363716125, 0.8480762243270874, 0.05178304389119148, -1.498441219329834, 0.6783933043479919, -0.16985756158828735, 0.06779367476701736, 0.6558824181556702, -1.5866525173187256, 0.19266216456890106, -1.0011810064315796, -0.34803131222724915, -0.04969791695475578, -0.14783304929733276, 0.02571273222565651, -1.0124644041061401, 0.6417440176010132], [-0.6332793235778809, -0.22075359523296356, -0.00411935755982995, -1.5561320781707764, -0.3377145230770111, 1.627104640007019, 0.5544494986534119, -0.633970320224762, 0.06975884735584259, 0.12223178893327713, 0.48241564631462097, -1.6160794496536255, 0.4805574119091034, -0.08331643044948578, -1.4346736669540405, 0.7498080134391785, -1.9573853015899658, 0.577915608882904, -0.6993177533149719, -0.4416085183620453, -0.8294968605041504, 0.2297355979681015, -0.044968120753765106, -1.112971544265747, 0.4629780054092407, 1.5431804656982422, -0.8901621103286743, 0.14733926951885223, -1.0914781093597412, -0.7467400431632996, -0.43421024084091187, 1.942908763885498, 0.4674094021320343, 0.02874569036066532, 0.7932663559913635, 0.7098630666732788, 0.7748125791549683, -0.8563192486763, -0.4571867883205414, 0.9802594780921936, -0.32248127460479736, -0.8691896796226501, -0.8895354866981506, -0.7693318128585815, 0.24542537331581116, 0.30219465494155884, 2.1001229286193848, -0.4171192944049835, 0.7897506952285767, 0.27682775259017944, -1.73721182346344, -0.08535581082105637, -1.0797464847564697, 0.18135403096675873, 0.7373015880584717, -1.839368224143982, -0.15574541687965393, -1.6278645992279053, 0.9549831748008728, -1.1002832651138306, -0.5931838154792786, -2.3769352436065674, 1.4496761560440063, 0.2631504535675049, -0.25470051169395447, -0.014666257426142693, -0.46259182691574097, 1.860703468322754, 0.00472854170948267, 1.0418306589126587, -0.017722567543387413, -0.22588381171226501, 1.4536558389663696, -1.1453007459640503, 0.1023230105638504, 0.9556251764297485, 0.29488644003868103, -1.0806128978729248, 1.09823477268219, 0.3624550700187683, 0.3626772165298462, -0.4363548457622528, -1.206549048423767, 1.7699612379074097, -0.22387604415416718, 1.0842422246932983, 0.27490106225013733, 1.2938475608825684, -0.48805397748947144, -0.13391366600990295, -0.08837154507637024, 0.7721623182296753, 0.05537132918834686, 0.927058219909668, 0.40626266598701477, -2.2786800861358643, 0.9353207349777222, -0.6695480942726135, 1.228882074356079, -0.10123280435800552, -1.872408390045166, 1.27480947971344, -0.8935869336128235, 0.8939937949180603, -0.3342178463935852, -0.28582414984703064, -0.5514557957649231, -0.5663473606109619, -0.42796850204467773, 0.4384222626686096, -0.35610923171043396, 0.31904295086860657, -1.9155669212341309, 0.12505067884922028, -0.9931938648223877, 1.288637399673462, 0.25260505080223083, -1.7252004146575928, 0.9411852955818176, -0.8530954718589783, 0.7937930226325989, -1.4300487041473389, 0.7514981627464294, -1.5050188302993774, -1.475351333618164, -0.4509761929512024, 0.1903836578130722, -0.7237801551818848], [0.06511497497558594, 0.6254827976226807, -0.19819824397563934, 0.7678884863853455, -0.7833388447761536, 0.8360174298286438, -0.23387406766414642, -0.4116545617580414, -0.6751323938369751, 0.562987744808197, 0.18732969462871552, -0.7167223691940308, -0.3764447867870331, 0.656701922416687, -0.013910742476582527, -1.1344659328460693, 1.6409764289855957, 1.140439510345459, -0.7521483302116394, 0.33553290367126465, -0.7232775092124939, -1.8280060291290283, -0.4661415219306946, -1.313020944595337, -0.9619213342666626, -0.5994630455970764, 1.4825879335403442, 0.39170539379119873, 1.2595317363739014, 0.8950271606445312, 0.13396945595741272, -0.18053396046161652, 0.8055582046508789, -0.4528767466545105, -1.2660202980041504, 1.0014792680740356, -0.8988728523254395, 0.00981485191732645, 0.2017381489276886, -1.4407137632369995, 0.6043292880058289, -0.022293340414762497, -0.3297429382801056, 0.9978141188621521, -1.0155843496322632, 0.06074083596467972, 1.12945556640625, 1.0043524503707886, 0.2822658121585846, 1.2481282949447632, -1.6778992414474487, 1.4991751909255981, -1.4973303079605103, -0.5339844822883606, -1.6236652135849, -0.3069484531879425, 0.4694592356681824, -0.0868934839963913, -1.6097787618637085, -0.707247257232666, 1.5539400577545166, -0.9260009527206421, 0.5864037275314331, -1.3842600584030151, -0.31872814893722534, -0.7214798927307129, -0.12452031672000885, 0.08098035305738449, -0.8065505623817444, 0.771299421787262, 0.20191584527492523, -1.7769044637680054, 0.059452902525663376, -0.7979507446289062, -0.5696141123771667, 0.7460594773292542, -0.5667790174484253, -0.8651301860809326, -0.12748456001281738, -1.16840660572052, 1.1573021411895752, -1.8675570487976074, -0.10737112164497375, 0.013989024795591831, 0.19212310016155243, -0.7823776006698608, 0.30447226762771606, 0.838681697845459, 0.531278133392334, -0.005686564836651087, -0.6590967774391174, 0.012973972596228123, -0.9223287105560303, 2.4412248134613037, 1.7552827596664429, 0.810283899307251, -0.27804693579673767, 1.1103270053863525, 1.4311699867248535, 1.260031819343567, 0.05282266438007355, -0.17617425322532654, 0.7893643975257874, 1.6416395902633667, 1.1260985136032104, 0.3963662385940552, 1.274039626121521, 1.5433801412582397, 1.5050288438796997, 1.3185151815414429, 1.6906110048294067, -0.8038648962974548, -0.41343197226524353, 0.757275402545929, 0.26891443133354187, -0.953060507774353, -0.3924361765384674, 1.0261372327804565, 0.07603480666875839, 0.20666173100471497, 0.5856690406799316, 0.5805749893188477, 0.2621476948261261, 0.7170840501785278, 0.3109821379184723, 1.8141283988952637, -1.810694694519043, 2.228405714035034], [-1.1362758874893188, 0.3678897023200989, -1.3064377307891846, -0.4925992786884308, 0.37970876693725586, 0.479969322681427, -1.6647626161575317, 0.41782230138778687, 1.1687917709350586, -0.9680095911026001, -0.6472963094711304, 0.03391721099615097, 1.6531420946121216, 0.15804076194763184, -0.8156555891036987, -0.3194001317024231, 1.70649254322052, 1.399165391921997, 0.5265073180198669, 0.06959927082061768, -1.5302966833114624, 0.3885796070098877, -0.9330306053161621, 0.5690267086029053, 0.4030163586139679, -1.7018238306045532, -1.1204302310943604, -1.3631829023361206, -1.589081883430481, 0.20871564745903015, 1.7258812189102173, 0.6630991101264954, 0.2224784642457962, -1.1029938459396362, 1.5717813968658447, -0.2154398411512375, -1.0059311389923096, 0.6080986261367798, -0.24417686462402344, -0.6594879627227783, 0.5490139722824097, 0.41925865411758423, -0.319892942905426, 1.7804012298583984, -1.215855598449707, -0.05688514560461044, 0.3509526550769806, -2.1584503650665283, 1.16481351852417, 1.1571732759475708, -1.1166791915893555, 0.25030824542045593, 0.4039127826690674, 1.6272767782211304, 0.7944691777229309, -1.3445932865142822, -0.39758777618408203, -0.6851399540901184, 0.4351361393928528, 0.07715371996164322, -1.071926236152649, 0.6288947463035583, 0.33518922328948975, -0.006405770778656006, 0.37604111433029175, -0.41778847575187683, -1.2618638277053833, 0.2526359260082245, -1.311418056488037, -0.08136120438575745, -1.479792833328247, 0.3988329768180847, -0.19869756698608398, 0.22589999437332153, -1.1311882734298706, 0.582630455493927, -0.06357347220182419, 0.2774955630302429, -1.466862678527832, -0.21583230793476105, 0.6388524770736694, -1.5676519870758057, -1.620383858680725, 0.22056689858436584, -0.5673993229866028, 0.8759351968765259, 0.0307453703135252, -0.679997980594635, 0.9398666024208069, -0.016392134130001068, -0.42748215794563293, 0.441387414932251, -0.15927737951278687, -1.2609076499938965, -1.2312294244766235, 0.5281439423561096, 1.7124367952346802, 1.2372488975524902, 1.3286559581756592, 0.45602861046791077, 1.6523661613464355, 0.5806316137313843, -0.5043410062789917, -0.860939085483551, 0.10809097439050674, -1.7416234016418457, 0.5108171701431274, -0.22018034756183624, -1.717821478843689, -0.34770283102989197, -0.4159304201602936, -0.0424702949821949, 3.0203192234039307, 0.11495718359947205, -0.2296125739812851, -1.7086095809936523, 0.8065909743309021, -0.9944584369659424, 1.3415366411209106, -1.2540364265441895, -0.49846309423446655, -1.3122308254241943, -0.9956842064857483, 1.9403934478759766, 0.3032655715942383, 0.8048362135887146, -0.4072796106338501, 1.354225993156433], [-1.8039952516555786, 0.9137102961540222, 0.23518776893615723, -0.7532379627227783, 0.2773289382457733, 0.9584349393844604, 0.4722519516944885, 2.663464307785034, 1.3835740089416504, -0.5069648027420044, -0.09692656248807907, -0.20390567183494568, 0.3722629249095917, -0.5376171469688416, -0.47156524658203125, 0.029620399698615074, 1.3194966316223145, 0.5070176124572754, -0.8366928100585938, -0.4336264729499817, 0.7332615852355957, -0.3560200035572052, 1.2399144172668457, 1.1421657800674438, 1.0433566570281982, -1.3860266208648682, 0.8379753828048706, 0.32280802726745605, -0.8311750888824463, -1.0495902299880981, -1.6686166524887085, -0.3995319604873657, 0.25498461723327637, 0.5645738244056702, 0.3394244611263275, 0.8714953660964966, -0.6258808970451355, 1.9286916255950928, 0.9202293157577515, -1.0293684005737305, -0.4974941611289978, 0.8345630764961243, -1.456628441810608, 0.5810046195983887, -0.27515092492103577, -0.30277219414711, 0.41354900598526, -0.4438358545303345, 0.409259170293808, -1.302463173866272, 0.40742531418800354, -0.09754369407892227, -1.1626302003860474, -1.5928115844726562, -0.3036458194255829, 1.7519844770431519, -0.19001619517803192, -1.1435816287994385, 0.22969940304756165, 0.9332537651062012, -0.31706660985946655, 0.8727025985717773, 0.8667135834693909, 0.3582113981246948, -0.5036813020706177, -0.3334202766418457, -0.24418771266937256, 0.13827982544898987, -0.4598901569843292, -0.43335843086242676, -1.9237060546875, 1.4216753244400024, 0.6569356918334961, -0.3976820409297943, 1.6846617460250854, 0.6265420317649841, -0.8734488487243652, 0.12685707211494446, -0.36503109335899353, -0.08337479829788208, -0.4605073034763336, -1.6245166063308716, 0.7009616494178772, 0.31888946890830994, -0.1009342223405838, 0.10843171924352646, -1.226060152053833, -0.06887315958738327, -0.03101101517677307, 1.5183771848678589, 0.2906263768672943, -1.1071346998214722, -0.15637169778347015, -0.3462308645248413, 0.6279187798500061, -0.49646684527397156, -1.2046918869018555, 0.5998539924621582, 0.9395920634269714, -0.9851274490356445, 0.2884275019168854, -0.07042990624904633, 1.779846429824829, 0.9492893815040588, -1.704058051109314, 0.22444704174995422, -1.069707989692688, 0.145823672413826, 1.4164412021636963, -0.9007242918014526, -0.8310040831565857, 1.1072567701339722, 1.4838217496871948, 1.6772854328155518, -1.2216449975967407, 0.118177630007267, -0.2612251937389374, -0.29649409651756287, 0.6538073420524597, 0.001500432495959103, -0.02690194919705391, -0.07965143024921417, -1.630527138710022, -0.2270069718360901, 1.3876049518585205, -0.5639526844024658, -1.1104495525360107, 0.6055490374565125], [-0.2514302432537079, 0.4076828062534332, 1.413799524307251, 0.2953275144100189, -0.4225691854953766, 0.3281460404396057, 0.6581742167472839, 0.7901555299758911, -0.8885655999183655, 0.30518898367881775, 0.6458919644355774, 0.08226275444030762, 0.14034925401210785, 1.3164021968841553, 1.7151395082473755, 0.9661809802055359, 0.7302525043487549, 1.345439076423645, -1.2651019096374512, 1.3751009702682495, -0.6649725437164307, 0.25176501274108887, 1.6668449640274048, -0.36498549580574036, -0.41403618454933167, -0.6422683596611023, -0.37870511412620544, 0.8864216208457947, -0.6261876225471497, -0.27835431694984436, 0.22057242691516876, -0.9487342238426208, 0.24422836303710938, 0.17050346732139587, -0.8079922199249268, -0.17600084841251373, -0.7640542387962341, -1.020065426826477, -0.41521209478378296, 1.093461275100708, -1.101949691772461, 0.7208095192909241, 0.9630027413368225, -0.4002901613712311, 0.573775053024292, 0.30970898270606995, -1.1120920181274414, 1.178383231163025, 1.4426573514938354, 1.9436148405075073, -0.5700956583023071, -0.44734033942222595, -0.24145832657814026, 0.3546142876148224, -0.3180212676525116, -0.8822001218795776, 0.37193313241004944, -0.8090246915817261, 2.8261656761169434, 1.8451120853424072, 0.06460494548082352, 1.229999303817749, -1.2578139305114746, -0.06509213894605637, 0.8906994462013245, 1.315298080444336, 0.7159479856491089, -0.766464352607727, -0.6809656620025635, 1.0438212156295776, -0.8807698488235474, -0.029305867850780487, -1.1451573371887207, 0.23712636530399323, 0.6836196184158325, 0.2883722186088562, 0.6227049827575684, 2.029327154159546, 0.7769667506217957, 0.28048476576805115, -0.11263148486614227, -1.4961267709732056, -1.0358679294586182, 0.17604559659957886, 0.664955735206604, -0.9488399624824524, -2.221204996109009, 1.6198539733886719, -0.5714133381843567, 1.5953936576843262, 1.453827142715454, 0.18182192742824554, 1.2040125131607056, -0.2599121034145355, 0.29736852645874023, 0.5689339637756348, 1.6636606454849243, -0.3110354244709015, 0.018359677866101265, -1.1766818761825562, -0.8069188594818115, 0.0590425543487072, 0.8205428123474121, 0.6728489995002747, -0.4818021059036255, 0.24869006872177124, 1.9574286937713623, 0.4553816616535187, 2.0392889976501465, 0.8271336555480957, 1.629084587097168, 0.6166686415672302, -0.5559636354446411, -0.13143402338027954, -0.515610933303833, 0.6797988414764404, 0.16118192672729492, -0.6239427328109741, 1.3278990983963013, -0.2454642653465271, -0.12648490071296692, 1.1751703023910522, -1.1808561086654663, -0.052101898938417435, -0.7599292397499084, 0.5248681306838989, 0.3528382182121277, -0.19657377898693085], [0.1454109400510788, 0.03234110400080681, -0.05656491592526436, 0.5456873178482056, -0.02677222713828087, 0.6106936931610107, 0.2574405074119568, -1.0573915243148804, -1.255033016204834, -0.7214955687522888, 0.9661771655082703, 0.7786125540733337, 1.6364009380340576, -0.7215110063552856, 0.2298966646194458, -0.7443014979362488, -0.5447147488594055, 0.16017377376556396, 1.1287301778793335, 0.6049492359161377, 0.06274951249361038, 0.9068374037742615, -0.0725053995847702, 2.000063180923462, -1.1298755407333374, 0.5161155462265015, -0.7607554197311401, -1.711510181427002, -1.1719547510147095, -0.05263933539390564, 0.4293544888496399, -1.8512073755264282, -0.5281819701194763, 1.3468458652496338, 0.8326259255409241, -0.3323451280593872, -0.18454806506633759, -0.42840880155563354, 1.2838164567947388, 0.8308687210083008, -1.1755868196487427, -0.30649811029434204, 0.12389247864484787, 1.986555814743042, -0.4062986969947815, 1.4021888971328735, -0.4283938407897949, -0.31930357217788696, -1.4376851320266724, 0.37653276324272156, 0.17661738395690918, -0.2673412263393402, -0.2540741562843323, 2.644080877304077, -0.014702304266393185, 1.6497479677200317, -0.5110073089599609, -1.3837471008300781, -0.09857312589883804, 0.7529525756835938, 1.3944871425628662, 0.5796160697937012, 0.4031241536140442, -0.407610684633255, 1.7114838361740112, 0.0772111788392067, -0.20928052067756653, 1.9937632083892822, 1.2207928895950317, -1.0511257648468018, -0.8460864424705505, -0.9830548167228699, -0.01879044808447361, -0.6276236176490784, -0.7656641006469727, 1.1275944709777832, 0.504551351070404, -0.014278477989137173, 1.4066331386566162, -0.14858850836753845, -0.3759588301181793, -0.33013424277305603, 0.8113874197006226, 1.1817094087600708, 0.6614319682121277, -1.334897756576538, 2.0108108520507812, -0.4126816689968109, -0.3653324246406555, 1.1300809383392334, -1.853506088256836, 0.6460473537445068, 0.1517719328403473, 0.7646132707595825, 0.5177063941955566, -1.5664255619049072, 1.1128218173980713, 2.466095209121704, 0.5115083456039429, -0.6681011319160461, 0.03153350576758385, -0.6202012896537781, -1.326961874961853, -0.17309284210205078, -0.902642011642456, 1.098430871963501, 1.056947946548462, -0.19253630936145782, -1.1646028757095337, -0.1710747927427292, 0.03793532773852348, 1.1622540950775146, 0.3759293556213379, -0.15587389469146729, -0.7683455944061279, -1.3976927995681763, -0.8206504583358765, 0.9651226997375488, -1.3603582382202148, 0.34894534945487976, -1.2111470699310303, -0.7071636915206909, -1.3923773765563965, -0.3078814446926117, 1.236769676208496, 0.09757710993289948, 0.17566880583763123, -0.7539161443710327], [0.4482976794242859, 1.5009031295776367, 0.5424546599388123, -0.985034704208374, -1.6390070915222168, -0.057205867022275925, -0.07372595369815826, 2.6311745643615723, 0.12911765277385712, -0.33051979541778564, 0.6976329684257507, 1.0132580995559692, -1.8477689027786255, -0.5090176463127136, 0.39788269996643066, -0.8178654313087463, -0.5294228792190552, 1.88243567943573, 1.521846055984497, 0.9150562286376953, -1.2872228622436523, -0.4088321030139923, -0.5866715908050537, -1.5507915019989014, -0.15127632021903992, -1.0739762783050537, -0.8870276212692261, 1.470355749130249, 0.2974534332752228, -1.5477510690689087, 1.470726728439331, -1.453963041305542, -0.47212207317352295, -0.5787002444267273, 1.7407950162887573, 0.33604371547698975, -0.485551655292511, 0.6299763917922974, -0.9719445705413818, 0.20592458546161652, -0.5182716846466064, -1.8653948307037354, -0.39280977845191956, -2.4498794078826904, 0.0065536703914403915, -0.47091901302337646, -0.36205577850341797, -0.45624488592147827, 0.2132415622472763, 0.6150904297828674, -0.520427942276001, -0.9821168184280396, -0.7369136810302734, -0.504872739315033, -0.8497076630592346, -1.1102336645126343, 0.5071865320205688, -0.032657917588949203, 1.2970176935195923, -1.3379712104797363, -0.7796623110771179, 0.9812655448913574, 0.05467107519507408, -0.982276201248169, -1.5922648906707764, 0.6670301556587219, 1.2546535730361938, -1.3623130321502686, 1.4523862600326538, 1.0746663808822632, 0.8233188986778259, 1.5345983505249023, 0.38905277848243713, 0.028871744871139526, 1.6456902027130127, 0.5574495792388916, -0.4870201349258423, -1.6414834260940552, 0.4331054389476776, 0.6659305691719055, 0.13635201752185822, -0.04843491315841675, 0.5553447008132935, 0.0921858698129654, -1.2695069313049316, 0.056321483105421066, -0.8697268962860107, -1.8016499280929565, -0.20300708711147308, 0.7515652775764465, 0.44360268115997314, -0.7785834074020386, -0.8375321626663208, -0.7939634323120117, 0.9593107104301453, -0.5379470586776733, -1.2582744359970093, 1.7260924577713013, 0.5710985660552979, 0.11414320766925812, 1.8830355405807495, 1.4821220636367798, 0.6184993982315063, 0.787998378276825, 0.38950878381729126, -0.47927504777908325, 0.5423405766487122, -1.0958786010742188, -0.01953326165676117, -0.8137505054473877, 0.44231557846069336, 0.6384865045547485, -1.0810757875442505, 0.893101692199707, -0.11420014500617981, -2.300583839416504, 0.3696352243423462, 0.30713099241256714, 0.7303984761238098, 2.0374233722686768, -2.083444595336914, -0.48679473996162415, -0.10698602348566055, -0.1000324934720993, 0.5086899995803833, -1.0034821033477783, -1.0973244905471802, -1.3685688972473145], [-1.598559856414795, 1.686253547668457, 0.8744266629219055, 0.3262680470943451, 1.4021968841552734, -0.9558638334274292, 0.4237912595272064, 1.5781474113464355, -1.5488767623901367, 1.5474191904067993, 0.20556381344795227, -2.570470094680786, -0.9917207956314087, 1.5271632671356201, 1.1519898176193237, -1.7388533353805542, -0.0868179127573967, -0.037785615772008896, 0.5618952512741089, -0.5833553671836853, -0.2707309126853943, 0.3532103896141052, -1.4310312271118164, -1.1471238136291504, 2.2453091144561768, -0.011338516138494015, -1.5587053298950195, 1.3739134073257446, 0.8483996987342834, -1.6939421892166138, -0.3438645899295807, 0.6793661117553711, 0.34622105956077576, 0.5124883055686951, 0.578399658203125, -0.5990514755249023, 0.8791600465774536, 1.0563572645187378, -0.3875914216041565, -0.10691382735967636, -0.9693209528923035, 0.4171864688396454, -0.6680940985679626, -1.165016531944275, 1.014747977256775, 0.6202700138092041, -1.0009682178497314, 1.0353596210479736, 0.6914085149765015, 0.3772064745426178, -1.4653536081314087, -1.0627552270889282, 1.2730629444122314, 1.867655634880066, 0.9540593028068542, -0.6665012836456299, -1.7674692869186401, -1.444352626800537, 1.3923877477645874, 1.4790815114974976, 0.7695518136024475, 0.6267125010490417, 0.2720564007759094, -0.17121398448944092, 1.0678030252456665, -0.9459783434867859, -0.5811198353767395, -0.3308764100074768, 0.48289307951927185, -0.512104332447052, -0.5889146327972412, -0.4628346562385559, -0.15221109986305237, 0.09432777017354965, 0.002105538034811616, -2.0437281131744385, 0.605347752571106, -1.2539472579956055, 0.5349598526954651, 0.09082642942667007, -0.05377054959535599, -0.5367770195007324, 0.7195292115211487, 0.3293517231941223, -1.1248047351837158, 1.072157859802246, -0.6470286846160889, -1.446679949760437, -2.7171030044555664, 0.8164474368095398, 1.9249650239944458, 0.6365463137626648, 0.23308207094669342, -0.38276609778404236, -0.35942044854164124, 1.1194641590118408, -0.3260347247123718, -0.5435194373130798, -1.6587892770767212, -1.0585572719573975, -0.17864933609962463, 0.08662913739681244, 0.7408085465431213, 0.22746101021766663, -0.11144690215587616, -0.6342819929122925, 1.0780075788497925, 0.4264613687992096, -2.405297040939331, 0.39033007621765137, -1.8333863019943237, 1.1766823530197144, 0.8991080522537231, 1.36064875125885, 0.59369295835495, 0.15799777209758759, -1.41949462890625, -0.9944881200790405, 0.16204722225666046, 0.04867030307650566, -0.5390655398368835, 0.8665722608566284, 0.372676283121109, -1.1406431198120117, 0.15966059267520905, -0.5433835387229919, 0.3327323794364929, 0.34346696734428406], [0.9259183406829834, 0.7118431329727173, 0.5534571409225464, 1.434684157371521, 0.4841296374797821, -0.5393239855766296, 0.4468880593776703, 0.08818421512842178, 0.17200197279453278, 2.778571844100952, -0.6922799944877625, -1.5020517110824585, -1.2131149768829346, 0.07000412791967392, -1.901006817817688, 0.1062418669462204, -0.8269566893577576, 0.04797693341970444, -0.47043633460998535, -1.0472742319107056, -0.31877338886260986, 0.2687033712863922, 0.5904529094696045, -0.08749736100435257, -0.5847625732421875, -1.3969554901123047, 0.3896213471889496, 0.6240916848182678, 0.0911022275686264, -0.9777604341506958, -1.037322998046875, 0.04474518448114395, 0.11634168773889542, -0.9083489179611206, -1.0095051527023315, 0.7774531841278076, -0.41654494404792786, -0.30932649970054626, -0.24908389151096344, -0.7811788320541382, 0.2450648546218872, -0.066081702709198, 0.9008489847183228, -0.5171324610710144, -1.2488977909088135, 0.4452006220817566, -0.3926044702529907, -1.2032997608184814, -0.7138754725456238, -1.1551612615585327, -0.4095667004585266, -0.28870415687561035, 0.7845638990402222, 0.32358318567276, -1.9806395769119263, -0.7869843244552612, 1.2515764236450195, -0.32912498712539673, -0.3261168897151947, 0.7500221729278564, -0.7496887445449829, -0.40365391969680786, -0.5016937851905823, -0.21468490362167358, -1.0262094736099243, 0.8068956136703491, 1.380515694618225, 1.3680503368377686, -1.647094488143921, -1.2393032312393188, -0.19225119054317474, 0.9162857532501221, 0.5008055567741394, -0.8106627464294434, -0.6959147453308105, 0.8633808493614197, 1.0603870153427124, 0.5220771431922913, 0.8816876411437988, -0.5242847204208374, 1.4118810892105103, 0.2240128517150879, 0.6137387752532959, -0.25810715556144714, 0.6253007054328918, 0.5421754717826843, -0.048464324325323105, 1.2801620960235596, -1.2594764232635498, 0.0037185573019087315, -1.2786511182785034, 0.6270976662635803, -0.3818625509738922, 0.2959659993648529, 0.6683269143104553, 0.372674822807312, 1.1329704523086548, 0.3614178001880646, -0.791882336139679, -1.2440218925476074, 0.27500060200691223, -0.03582314774394035, 0.40332016348838806, -0.2172376811504364, -1.3167608976364136, -0.25893357396125793, 1.151999592781067, -0.14286063611507416, 1.004126787185669, 0.7005105018615723, -0.44712215662002563, 0.1997062861919403, -0.7065734267234802, 2.019775390625, 1.2307329177856445, 0.40727853775024414, 2.016864061355591, -0.2097845822572708, 0.721462607383728, 0.640723466873169, -1.099774718284607, 0.38374584913253784, -0.2769943177700043, 0.2687171995639801, 1.3581223487854004, 0.32460588216781616, -1.7981359958648682, -0.5410029888153076], [-0.6293008923530579, -1.4370297193527222, 0.33332785964012146, 0.10734037309885025, -0.3273523151874542, 0.5266620516777039, -1.9416823387145996, -1.017484188079834, -0.3046625852584839, 1.3243038654327393, 0.18011470139026642, 0.7673693895339966, 0.8980967998504639, -0.1356389969587326, -1.675157904624939, -0.20584948360919952, -0.07512824982404709, -0.8695507049560547, 0.7389235496520996, 0.9047579169273376, -1.9415299892425537, -1.7907222509384155, 1.600778341293335, -0.3744044005870819, -0.8365116715431213, 0.9552608728408813, 0.2194904386997223, 0.7980625629425049, -1.4349167346954346, 0.1334126740694046, -0.6622114181518555, -0.6569992899894714, -0.842120349407196, -0.6047045588493347, -1.6781196594238281, 0.09798774868249893, 0.1332244575023651, 0.1259796917438507, -1.7804003953933716, -0.0801805704832077, -0.2068757861852646, -0.06217969208955765, 0.05574282631278038, -0.7841144800186157, -0.08841858059167862, 0.19287097454071045, 0.38550496101379395, -0.6383222937583923, 0.11719398945569992, 0.46532878279685974, -0.2573550343513489, -2.0419249534606934, -0.0574374720454216, -1.409852385520935, -1.012234091758728, -1.28269362449646, -0.48672324419021606, 0.7414838075637817, -0.45379310846328735, 0.5868746042251587, -0.6975628733634949, 0.5635294318199158, -0.8614237904548645, 0.06878393143415451, -0.6248432993888855, 0.156193807721138, -0.6376727223396301, 0.7179900407791138, -0.9195365905761719, -0.09224535524845123, 1.1191375255584717, 0.9989290833473206, 1.291587233543396, -0.11694569140672684, 0.9601341485977173, -0.17763136327266693, 0.1671776920557022, 1.661508321762085, -1.0717322826385498, 0.8645772337913513, 0.8745611310005188, -0.3927841782569885, 0.74718177318573, 1.1072957515716553, -1.5520938634872437, 0.9937694668769836, 0.4265202581882477, 1.3408639430999756, -0.18583722412586212, -1.201027512550354, 0.46535539627075195, 0.33914318680763245, 0.7739090323448181, -0.20703865587711334, -1.1117422580718994, 0.17349916696548462, -0.4883281886577606, -0.8766536712646484, -0.1480531096458435, -0.5173358917236328, -0.46620944142341614, -0.26756349205970764, 1.1261861324310303, -0.5427426099777222, -0.472920298576355, 0.2959213852882385, 0.5416566729545593, 1.8837915658950806, -0.826188862323761, 0.6302873492240906, -0.9717553853988647, 1.5577958822250366, -0.6943698525428772, 1.498167634010315, 0.5438780188560486, -1.785147786140442, 0.7273029685020447, -0.3063662648200989, -0.019053438678383827, -0.5238338708877563, 2.340392589569092, 1.1387569904327393, 1.6742992401123047, 0.29252538084983826, -0.48015066981315613, -0.4755500555038452, 0.38926878571510315, -1.210754156112671], [0.18811216950416565, -1.4471333026885986, -0.42451873421669006, -0.2578589618206024, -0.15150491893291473, 2.834381341934204, -0.0309564471244812, -1.2994019985198975, 0.8531467318534851, -1.6298856735229492, 0.028090571984648705, 0.857991099357605, 1.5469577312469482, 0.9047651886940002, -1.0780811309814453, 0.9061495065689087, -0.011637510731816292, -0.14407968521118164, 1.2981334924697876, -0.3989194929599762, 0.1097443550825119, 0.17161934077739716, -0.8167262673377991, -1.8012200593948364, 1.7916231155395508, -0.0473911426961422, 0.9819075465202332, -0.5095195770263672, 0.29981690645217896, -2.305527448654175, 1.0547618865966797, 0.3501361012458801, -1.5534749031066895, -0.06654069572687149, -0.9650412201881409, 0.7652509808540344, -1.3661928176879883, -0.7364485859870911, 0.7936291098594666, 1.665554404258728, 1.7800590991973877, -0.6662124991416931, 1.4802029132843018, 0.2987017333507538, 0.7082152366638184, 0.35260993242263794, -0.448290079832077, -0.19520898163318634, 0.5626929402351379, 0.018791891634464264, -2.0653345584869385, 0.7873831391334534, -0.07517211884260178, -0.3437696099281311, 0.8857041001319885, -1.0272241830825806, -0.5718029737472534, -0.9606199264526367, 0.2813911437988281, 0.5612221956253052, -0.7461362481117249, -0.5838039517402649, 0.5800950527191162, -0.3788686692714691, -0.8297980427742004, 0.5058175921440125, 0.5034040808677673, 1.7929600477218628, 0.28758740425109863, 0.6524497270584106, -1.3546409606933594, -0.29332223534584045, 0.36257168650627136, -0.13820479810237885, 0.9307807683944702, -0.7364405989646912, 0.34559929370880127, -0.9948930144309998, 1.685731053352356, -0.15324918925762177, 1.04022216796875, -0.2953300476074219, 0.10932473093271255, -0.4638964533805847, -1.1291134357452393, -0.1709839105606079, 1.3505749702453613, -2.0640556812286377, -1.190125823020935, -1.0400400161743164, -1.1919941902160645, 0.879435658454895, -1.1462578773498535, 0.9699497818946838, -0.025301475077867508, 0.930884063243866, -0.5672661066055298, -1.4233994483947754, -0.5185144543647766, -0.7638878226280212, 0.47368496656417847, -1.1262515783309937, 0.6853868961334229, 0.3755730092525482, -0.03190240263938904, 1.0457912683486938, 0.9656731486320496, -0.4250009059906006, 2.667574167251587, -0.05161212012171745, -3.0497355461120605, 0.014133259654045105, -1.1051818132400513, -0.21660038828849792, -0.8118808269500732, -1.248081922531128, 1.5960438251495361, 1.5676488876342773, 0.80870121717453, -0.28660574555397034, 0.5146308541297913, -0.1687842607498169, 0.16063128411769867, -0.6658499240875244, 1.3368308544158936, -0.7603154182434082, -0.5157249569892883, -1.3392736911773682], [0.6311339735984802, -0.7633257508277893, 1.28204345703125, -0.06337399780750275, -0.03818228468298912, 0.8548281788825989, -0.8771178126335144, 0.009301171638071537, -1.659010887145996, -0.4686201512813568, -0.22723418474197388, -0.21292059123516083, 0.5093296766281128, 0.7847908735275269, -1.554277777671814, 0.34578484296798706, 1.0818885564804077, 0.6664729118347168, 1.5606474876403809, -0.10663309693336487, -1.0288455486297607, -0.9117984175682068, -0.2521722912788391, -0.8153417706489563, 1.8523601293563843, -0.6685522794723511, -0.5983830094337463, -0.33402618765830994, 1.8283659219741821, 0.19399355351924896, -0.8460999727249146, -0.4523298442363739, -0.3501763641834259, 0.9576064348220825, 1.8524234294891357, 1.045060157775879, -0.06592300534248352, 1.1414241790771484, -0.20384563505649567, -1.2437376976013184, 0.8784591555595398, 1.6578435897827148, 1.0748885869979858, 0.07297723740339279, 0.8839971423149109, -0.20257876813411713, 1.3421818017959595, 0.9263454079627991, 0.2949073016643524, 1.2602607011795044, -0.1873759925365448, -0.8659796118736267, 0.15043970942497253, 0.49572524428367615, -0.4072948396205902, -0.9984647631645203, 0.09989997744560242, 0.49837973713874817, 0.4292135536670685, -2.521153688430786, 0.45940130949020386, -0.5992996096611023, 0.20861199498176575, 2.1467373371124268, -1.4204559326171875, -1.2410722970962524, 0.6043877005577087, -0.1341336965560913, 1.0819036960601807, -0.050316184759140015, -0.5774322748184204, -0.17493128776550293, 1.1175239086151123, -2.3988797664642334, -0.94049072265625, 0.24844709038734436, 0.6198724508285522, 0.8213196992874146, 1.3128342628479004, 0.9455381631851196, 0.3578037619590759, -1.151244878768921, -1.1144119501113892, -1.9540841579437256, -0.6185871958732605, -1.7147071361541748, -1.22059166431427, -0.16942903399467468, -0.3859158754348755, -1.0255978107452393, -0.12124817818403244, 1.1847823858261108, -0.492804616689682, 0.12527209520339966, 0.03605371341109276, -1.0016628503799438, 0.38090401887893677, 0.06944812834262848, -0.8802851438522339, -1.0652358531951904, 0.6873892545700073, 0.027320297434926033, -0.13912546634674072, 0.012866218574345112, -0.06490755081176758, 1.2235264778137207, 0.9295039176940918, -0.13219314813613892, -0.22006161510944366, 0.6580455899238586, -1.1339436769485474, 0.9034184813499451, 0.48923367261886597, -1.6867916584014893, 0.2990156412124634, -1.1256126165390015, -0.09257760643959045, 1.138999104499817, -0.4474901258945465, -1.6268426179885864, 0.8728852272033691, -0.9989661574363708, 0.003190798219293356, 0.6311021447181702, -0.014504186809062958, -1.7752100229263306, 0.22707980871200562, 0.816156804561615], [-1.0544480085372925, 0.17644494771957397, 0.952677845954895, 0.49736499786376953, -0.7248064279556274, 2.3339741230010986, -0.6642277836799622, 1.1079680919647217, 0.16825519502162933, -0.3274517059326172, -0.8190571069717407, -0.9436972141265869, 0.28158944845199585, 0.35440006852149963, -0.01539335586130619, -0.29057127237319946, 0.6149653196334839, 0.9877046942710876, -0.5218161940574646, 0.7429559230804443, -1.6978416442871094, -3.1095855236053467, -1.122787356376648, 0.6390895247459412, 0.6353304982185364, 0.22389496862888336, 0.04928996041417122, 0.4793466329574585, -0.36219432950019836, 2.7780370712280273, -0.07804779708385468, -0.20958809554576874, -1.1390939950942993, -1.6000561714172363, -0.4553276598453522, 1.0606074333190918, 1.8492106199264526, 0.14075712859630585, 1.7502071857452393, -0.7270490527153015, 1.5358537435531616, -1.5504125356674194, 0.09156522899866104, 2.401634693145752, -0.47386831045150757, 0.8089704513549805, -0.2250405251979828, 0.6833785772323608, 0.9399593472480774, 0.5460250377655029, -0.44516703486442566, 0.3734169900417328, -1.6427085399627686, 0.014277464710175991, 0.27699244022369385, 0.662193238735199, 1.5682183504104614, 0.0062767984345555305, 0.21052737534046173, 1.2487823963165283, 1.3117969036102295, 0.1707635521888733, 1.0129508972167969, 0.6713680028915405, -0.7736542820930481, 0.436452716588974, 0.15957742929458618, -0.06228366866707802, 1.6836494207382202, -0.9008969068527222, -1.6459972858428955, 0.6283133029937744, -1.240182876586914, -1.4896985292434692, 1.049783706665039, 1.6370846033096313, -0.8075161576271057, -2.425410509109497, 1.5270971059799194, 0.4608045220375061, -0.08358390629291534, 1.7394107580184937, -1.0595563650131226, -1.5188528299331665, -0.997823178768158, -0.03279886394739151, -0.3293623626232147, -0.5058444738388062, -0.35041195154190063, -0.6144862771034241, 0.32106897234916687, 1.6443889141082764, 0.7347113490104675, 0.12601523101329803, -1.0392531156539917, 0.6637342572212219, 0.11838482320308685, -0.9635063409805298, -0.4269667863845825, -0.827922523021698, 1.5175671577453613, -0.5440993905067444, -0.4733831286430359, 1.1595927476882935, 1.4641462564468384, 0.387389212846756, -0.07002852112054825, -0.3923927843570709, 1.1742299795150757, -0.12921899557113647, 0.12441155314445496, 0.4280894994735718, 1.1439343690872192, -2.1990017890930176, -1.9313855171203613, -1.3007512092590332, -0.5408768653869629, -0.188572958111763, -0.4477357864379883, 0.10072483122348785, 0.9920734763145447, -0.21839193999767303, 0.08119985461235046, -0.5134326815605164, 1.9810644388198853, -0.6428413391113281, 0.3341660797595978, -0.2790757715702057], [-1.4549189805984497, -1.1256285905838013, -0.009370354004204273, -0.44186466932296753, 0.3016379177570343, -0.7430200576782227, -1.1543782949447632, -0.011060258373618126, 1.7170133590698242, 0.12169286608695984, 0.2533986568450928, 1.0786476135253906, 0.03125365450978279, -0.32540854811668396, 0.5003007650375366, -1.6081547737121582, -0.7164481282234192, 1.7150168418884277, 0.6871266961097717, 0.5216938257217407, 0.06431645900011063, 0.7026134729385376, 0.28292691707611084, 0.7627120018005371, 2.0509891510009766, -1.1243137121200562, -0.5426804423332214, 0.5779951214790344, -1.4915658235549927, -1.121579647064209, 1.4077913761138916, -0.13309313356876373, 0.6911161541938782, -1.0068089962005615, -1.2643145322799683, 0.2541302442550659, -1.0976057052612305, 0.6048778295516968, 0.19257399439811707, -0.4975520372390747, 0.5982038974761963, -0.8629299998283386, 0.24435491859912872, -0.17398416996002197, -1.1687588691711426, 1.3472003936767578, -0.19603587687015533, 1.8385639190673828, 0.0756257027387619, -1.0076478719711304, -0.5035281181335449, 0.9898690581321716, -1.3590283393859863, -1.1117902994155884, 0.4631657004356384, 1.063149094581604, -0.3569880425930023, -0.4155973792076111, 0.6184189319610596, 0.3190285265445709, -0.23509518802165985, -0.2800166606903076, 0.29188504815101624, 0.9399711489677429, -2.242769956588745, -1.7440088987350464, -0.8128805160522461, 0.7538555860519409, -1.479915976524353, 1.061625599861145, -1.3086767196655273, -0.9313240647315979, 0.607093334197998, 0.0653604194521904, -0.5130376815795898, 1.3624969720840454, -0.9331185221672058, -0.2780715823173523, 0.3738894462585449, 1.175091028213501, -0.7469993829727173, 0.026344018056988716, 0.272223562002182, 0.7565047144889832, 1.0109690427780151, -0.974787175655365, 0.6003850698471069, 0.6419731378555298, -0.2530260384082794, -0.14485523104667664, 1.5816466808319092, 1.4388887882232666, 1.0422277450561523, -0.755939781665802, 0.16680945456027985, -0.2530224025249481, -0.0008377619087696075, -0.2851778566837311, 0.606465220451355, -0.35184693336486816, 0.9169138073921204, 1.246271014213562, -0.6995730400085449, -0.4832107126712799, 0.042225394397974014, -0.9194362759590149, -1.7696717977523804, 0.48012733459472656, -1.1058145761489868, 0.10717513412237167, -0.41989830136299133, -0.927497386932373, 1.7690527439117432, 0.4241352081298828, 0.03190242871642113, 0.9298111796379089, 1.0674593448638916, 0.4475841224193573, 0.10776443779468536, -0.20595118403434753, 0.16915248334407806, 0.8117578029632568, 1.2571333646774292, -1.176095724105835, -0.2959970235824585, 0.4071594476699829, -1.2522332668304443, 0.1711837351322174], [-0.5218031406402588, 0.7029337882995605, -0.0924559086561203, -1.8751921653747559, 0.22469480335712433, 0.44833865761756897, -2.1902260780334473, -1.0093272924423218, 1.1088347434997559, 3.0334150791168213, 0.38715776801109314, 0.26063868403434753, -1.9177716970443726, 0.9957942366600037, -1.9517536163330078, -1.3533216714859009, 0.7350893616676331, -0.4993475377559662, -0.0013191510224714875, 0.5842685699462891, -0.06992598623037338, 1.557641625404358, -1.1018600463867188, -1.3700788021087646, -0.6515282392501831, -0.09072330594062805, -0.0947342962026596, 0.5236667394638062, -0.32224705815315247, 0.2911159098148346, -0.42490315437316895, 0.7027931809425354, -0.490786612033844, -0.13865433633327484, 0.24061211943626404, 0.3071759343147278, -0.42324429750442505, -1.311745285987854, -0.22935238480567932, 0.14229334890842438, 0.37996700406074524, 1.847143530845642, -0.6279035806655884, -1.6866756677627563, 0.3840333819389343, 0.9813510775566101, -0.1476452648639679, 1.6376726627349854, -0.477425754070282, -0.10541899502277374, 0.4293186068534851, -0.293729692697525, -1.6291940212249756, 0.8086956739425659, 1.0383104085922241, 1.3840148448944092, -1.6312503814697266, -0.39930227398872375, -0.0244719460606575, 1.1948904991149902, 1.5052438974380493, -1.1596428155899048, -0.28777647018432617, -0.07000721991062164, 0.1003948450088501, 1.2480744123458862, 0.1959535926580429, 0.23663020133972168, -1.6206233501434326, 0.03775635361671448, -0.15945874154567719, -0.4924223721027374, 0.4168892204761505, 0.28565338253974915, -0.8615715503692627, -0.3005092740058899, -1.738120198249817, -0.8494449257850647, 0.06307079643011093, -0.31204190850257874, 1.3630603551864624, 0.050658147782087326, -2.9359052181243896, -0.14797039330005646, 1.1120963096618652, -1.506620168685913, -0.04201788827776909, 1.6695454120635986, -2.4717278480529785, -0.44685637950897217, 0.32188680768013, 2.307281970977783, -0.6958922743797302, 0.3541318476200104, 1.2043012380599976, 1.6803609132766724, 0.8151023983955383, -0.38748443126678467, -0.21126116812229156, 1.6650025844573975, -2.4882655143737793, -0.24721753597259521, -0.6358664035797119, -0.05110330507159233, -0.022110160440206528, -0.29821309447288513, -0.26487669348716736, 0.5212991833686829, 0.22421535849571228, 1.4218817949295044, -1.544042706489563, 0.9202598333358765, -0.95382159948349, 0.8192780017852783, -1.0893782377243042, -0.7868313193321228, -0.7021535634994507, -0.891408383846283, -0.31062328815460205, -0.28475698828697205, 1.9150997400283813, 1.0121196508407593, 0.48441535234451294, -0.3124716877937317, -0.24486792087554932, -0.6113556623458862, 1.3820416927337646, -0.6444088816642761], [-0.4461683928966522, 0.48133689165115356, 0.9994179010391235, 0.005227846559137106, 0.07497463375329971, 0.5359443426132202, 0.4850466251373291, 0.07198154926300049, -1.3300079107284546, 1.3560092449188232, -1.4491077661514282, -0.12240162491798401, 1.3107274770736694, 1.0530930757522583, -1.282724380493164, 1.1959595680236816, -0.4900275766849518, 0.09451381117105484, 0.055689290165901184, 1.055598258972168, -0.24837133288383484, 0.4029950797557831, 1.118203043937683, -2.2679519653320312, 0.8972423672676086, 0.33516231179237366, 0.36888134479522705, 1.3391766548156738, -1.102660059928894, 1.4366333484649658, -1.1999002695083618, -0.39660975337028503, -0.2042863517999649, -1.2695276737213135, 1.0529130697250366, -0.5393415093421936, -1.977135419845581, -0.11312029510736465, -1.2674800157546997, -0.22630009055137634, -1.0497221946716309, 0.3205786347389221, 1.1324962377548218, 0.06610272824764252, 0.5234226584434509, -0.2108222097158432, -0.19496822357177734, 1.6790496110916138, 0.5302493572235107, -0.6446993350982666, 1.0888733863830566, 0.9201539754867554, -0.661228358745575, 1.856224775314331, -1.2196999788284302, -0.597543478012085, 0.11059368401765823, -1.2875163555145264, -0.6128575801849365, 0.9929986596107483, 0.6084112524986267, 0.6476020216941833, -0.5088300704956055, -0.30417802929878235, 0.4560590386390686, -0.13338957726955414, -1.454841136932373, -0.33311381936073303, -0.013907371088862419, 1.0588977336883545, -0.8588242530822754, 0.5698864459991455, -0.7059193849563599, -0.734972357749939, -0.06962230056524277, 2.9117205142974854, -1.544778823852539, -1.7639552354812622, -1.9778594970703125, -0.33308878540992737, 1.1970313787460327, -2.0582079887390137, 2.009566307067871, 0.5664591789245605, -2.1571402549743652, 0.24648813903331757, 1.4883906841278076, 0.06552302837371826, 0.9197423458099365, 0.7379955053329468, 0.5376917719841003, -0.6094271540641785, 1.051606297492981, 0.3235889971256256, -0.14803287386894226, -1.9224324226379395, -1.7515883445739746, 0.21542705595493317, -0.7049407362937927, 0.06960161030292511, -0.10050518810749054, -1.6819742918014526, -1.4259588718414307, 0.030590977519750595, -1.1625337600708008, -0.2241659015417099, 0.08726520836353302, 1.1021924018859863, 0.16064453125, 1.0405739545822144, 0.08517450094223022, 0.057995300740003586, -0.49616754055023193, 1.7012393474578857, -0.508965253829956, 0.2146279215812683, -1.3040400743484497, 0.4424978196620941, 0.98827064037323, 0.18801121413707733, 0.4802938997745514, 1.0411604642868042, 0.8226026892662048, -1.2351006269454956, -0.5061911344528198, -0.21877501904964447, 0.16736307740211487, -0.8587132096290588], [-1.6709251403808594, 1.6397705078125, -0.6160966157913208, -0.6684243083000183, 1.0285311937332153, -1.9787218570709229, 0.7870346307754517, -1.053165316581726, 0.5243290066719055, 0.410550057888031, -1.8644813299179077, 0.2565722167491913, 1.7558835744857788, -0.34912851452827454, 0.07699575275182724, -0.41484537720680237, 0.05218590423464775, 1.0829583406448364, 1.0040515661239624, -0.6516132354736328, -0.747721254825592, -1.521407961845398, 0.5369576811790466, -0.41190385818481445, -2.2174344062805176, -1.2928948402404785, 1.7719732522964478, -1.3228850364685059, -0.8926188945770264, -1.0127732753753662, -0.9031786918640137, 0.6930099129676819, 0.9914812445640564, 0.897021472454071, -0.03851349279284477, 0.17792202532291412, -0.4435156285762787, -0.23105449974536896, -0.6354948282241821, -0.37275347113609314, -0.7976747751235962, -1.2658089399337769, -0.36884254217147827, -1.7059903144836426, 0.2958550751209259, -3.00527024269104, 0.25202131271362305, 0.33094391226768494, 0.2785080373287201, -0.06011855974793434, -0.8610068559646606, -1.694321632385254, 0.28319838643074036, 1.615209698677063, -1.5996549129486084, 0.4742780327796936, 0.22115297615528107, -1.0611029863357544, -0.5386652946472168, -0.07959683239459991, 0.7740404009819031, 1.7925175428390503, 1.826237678527832, -0.7363994121551514, 0.2903333902359009, -0.26098981499671936, -0.45545250177383423, -1.2519221305847168, 0.5269181132316589, -1.5242464542388916, 1.1406800746917725, -0.3240857720375061, -0.10446605086326599, 0.3098739683628082, -0.5650490522384644, -0.2679286599159241, -0.6409469842910767, 0.7543034553527832, 1.192604660987854, 0.3604643940925598, -0.06690935045480728, 0.3660551607608795, -0.905119776725769, -0.029951859265565872, 1.0934031009674072, 0.06975952535867691, 0.46098437905311584, -2.2840464115142822, -0.13483703136444092, -0.35484224557876587, 0.8415235877037048, -0.9110575318336487, 2.2951600551605225, 0.34241506457328796, -0.435656875371933, 1.0884931087493896, -0.27807798981666565, 0.5150513052940369, -1.1932857036590576, -0.16614685952663422, 1.8326780796051025, 0.4279382526874542, 1.2393521070480347, 0.7207130789756775, -0.38994935154914856, -1.6412544250488281, 0.28155970573425293, 1.380718469619751, -0.5487604141235352, 1.146636724472046, 0.9992477297782898, 0.43610161542892456, -0.2878299653530121, 0.5513497591018677, 0.5535387992858887, 2.0862650871276855, 1.6172466278076172, -1.093319296836853, 0.6103062629699707, -1.6978156566619873, -0.5227369070053101, 0.25456807017326355, -1.0942018032073975, -0.5259488821029663, 0.009124789386987686, 0.3185466527938843, 0.23669768869876862, 1.063256025314331], [1.2307952642440796, 0.13517709076404572, -1.1334068775177002, -0.37130501866340637, -1.1727076768875122, -0.6816905736923218, 0.6922377347946167, 0.617904782295227, -0.7105311155319214, -0.3235861361026764, 0.47742223739624023, -0.23642529547214508, 0.1353342980146408, -0.7348877787590027, 0.5966951847076416, 0.07016590982675552, 2.1031272411346436, 0.025654690340161324, 0.4435275197029114, 1.6156296730041504, 0.6281968951225281, 0.9811094999313354, 0.6571848392486572, 0.7424235939979553, -0.5296711921691895, -0.3360752761363983, -0.038690902292728424, 0.7168195247650146, 1.0565358400344849, -0.08613140136003494, 1.3303043842315674, 1.216676115989685, -0.9349687695503235, -0.1842900961637497, -0.11371252685785294, 1.166882038116455, -0.02808755822479725, -0.30811265110969543, 0.7611427903175354, 0.5137409567832947, -0.1495320349931717, 1.260188341140747, -1.2395517826080322, -1.7313796281814575, -0.26182156801223755, 2.188624143600464, -2.2084524631500244, -0.4721459746360779, -0.7837260365486145, -1.7643688917160034, -0.17129193246364594, 1.0852735042572021, -1.9077085256576538, 1.5323189496994019, -0.8061941862106323, -0.042092107236385345, 0.6761424541473389, -1.4520926475524902, -0.02711988054215908, -1.2735923528671265, 0.25231650471687317, 0.29952070116996765, -0.02575622871518135, 0.09034159779548645, -1.2560551166534424, 1.582293152809143, 0.42694151401519775, -0.18413542211055756, -0.16641725599765778, 0.43185245990753174, 1.1443486213684082, 1.572059154510498, -0.9718773365020752, -1.3364707231521606, -0.34704530239105225, -0.930569052696228, -2.1685290336608887, 0.49530553817749023, -0.5054472088813782, 0.6386947631835938, -0.2807788550853729, -0.6375459432601929, -0.7460227608680725, -0.7209078073501587, -0.025467809289693832, 0.04489052668213844, 2.4602103233337402, 1.048534870147705, -0.3770066201686859, 0.8096867203712463, -0.1531146615743637, -0.9788652062416077, 0.24658751487731934, -1.6792192459106445, 2.928609609603882, 1.140606164932251, -0.7159045338630676, 0.3979716897010803, -1.23239266872406, -0.3927546441555023, 1.0430649518966675, 1.1039377450942993, -0.9606306552886963, -0.8153195977210999, 0.40222638845443726, -1.6600487232208252, -0.2280939370393753, -1.2994939088821411, 0.47265589237213135, 1.1844242811203003, -2.5407862663269043, -1.065382719039917, -0.04921959340572357, 2.107060194015503, 1.5318423509597778, -0.5958754420280457, -0.6064426898956299, 0.09571082144975662, 0.3906711935997009, 2.8598015308380127, 1.5677541494369507, 0.13541576266288757, -0.797924280166626, -1.0184534788131714, 0.4927235543727875, -1.3303394317626953, -0.21661685407161713, 1.045657753944397], [-1.3136625289916992, 0.5410088300704956, 0.6989359855651855, 1.9146158695220947, 0.9674168825149536, -0.549312949180603, -0.1627204269170761, 0.6883724331855774, -0.01648438163101673, -0.5235732197761536, 0.8860735893249512, 0.8871996402740479, 0.7350234389305115, -0.051340702921152115, 0.5437618494033813, -0.7358782887458801, -0.14399750530719757, 1.2767632007598877, 0.2792837917804718, 0.03966839984059334, -0.27165448665618896, 0.8198856711387634, -1.3895771503448486, 0.27277034521102905, 0.22878213226795197, 1.0805259943008423, -0.5887590646743774, 0.8311144709587097, 0.7746235132217407, -1.2266603708267212, -0.14143206179141998, 0.07840505987405777, 0.21460849046707153, -0.6506487131118774, -0.4215969443321228, 0.5237174034118652, -0.46121692657470703, -0.7299726605415344, 0.32585233449935913, -0.3155786693096161, -1.4919489622116089, 0.35308384895324707, 2.244600296020508, 1.9256647825241089, 1.3720788955688477, 1.624679684638977, -0.1776675134897232, 0.6982225179672241, -0.9410890936851501, -0.3358718752861023, -1.4466242790222168, -0.3920212388038635, -1.8536313772201538, 0.40434542298316956, 0.33730292320251465, -0.6861308217048645, -1.3748674392700195, 1.9168177843093872, -0.9366881847381592, 0.19578586518764496, 1.448167085647583, -1.1086821556091309, -0.05083290860056877, 0.8468475341796875, 1.278215765953064, 2.029949426651001, -1.18468177318573, -0.23677456378936768, -1.0920300483703613, 1.2225096225738525, 0.506441593170166, 1.0382989645004272, 0.03498884662985802, -1.286727786064148, -0.8114867210388184, -0.5148258209228516, -1.7737351655960083, 0.26540887355804443, -0.09987278282642365, -0.02292739227414131, -0.9691578149795532, 0.14312537014484406, -0.010365457274019718, 0.4848315417766571, -0.8852823376655579, -1.4248510599136353, -0.573823869228363, 1.1597402095794678, -0.9448297023773193, 0.5256754755973816, -0.6238406300544739, -0.6985812783241272, -0.7658742070198059, 0.11812643706798553, -0.4629691541194916, -1.067426085472107, 2.0234036445617676, 1.5631792545318604, 1.053248405456543, -0.6178939938545227, -0.1337282806634903, 2.438591718673706, 0.32413890957832336, -1.344443440437317, 0.5345983505249023, 0.5256394743919373, -0.5765377283096313, 0.0754573792219162, -0.3977574408054352, 0.722846269607544, -0.7920037508010864, 0.3717104196548462, -0.6426270604133606, -0.5344972014427185, 0.9738653302192688, 1.2633901834487915, -1.675162672996521, 1.173163890838623, 0.10604798048734665, 0.9906370639801025, -0.7397177219390869, -0.11791294068098068, -0.6221038103103638, -0.15281352400779724, 0.15521498024463654, 1.7357937097549438, 0.05861053615808487, -1.122954249382019], [0.0919249951839447, 1.5785726308822632, 0.016953187063336372, -0.5744251608848572, -1.0151667594909668, -0.27755770087242126, -0.6193227171897888, 0.40666431188583374, -1.4573309421539307, 1.131597638130188, -0.19021472334861755, 0.7514354586601257, 1.1360899209976196, 1.253313422203064, 0.6468419432640076, 0.8845764994621277, 0.34633421897888184, 1.6095682382583618, -0.7853432297706604, -0.20140720903873444, -0.2106168568134308, 0.3183762729167938, 0.9834615588188171, -1.033130407333374, -1.6522157192230225, 0.8806320428848267, -0.5958388447761536, -0.987492561340332, 0.43761566281318665, -1.018431544303894, 1.6127595901489258, 0.16771860420703888, -0.4530109167098999, 0.8623137474060059, -2.1692848205566406, 0.08818525075912476, 1.472852349281311, -0.5335259437561035, 0.791862964630127, 1.0574641227722168, 0.6414585113525391, -0.7817811965942383, 0.9603762030601501, 1.0347275733947754, -1.1128120422363281, -0.25250735878944397, -0.44560423493385315, -1.2810560464859009, 0.6732425689697266, 0.20464615523815155, -1.6659656763076782, -1.9986579418182373, -0.052361950278282166, 0.15335793793201447, 0.3806452751159668, 0.08217493444681168, -0.27194878458976746, -1.6026828289031982, 0.40187695622444153, 1.1624058485031128, -0.06192764639854431, -0.8470535278320312, -1.3052852153778076, -0.10401541739702225, -0.34500253200531006, 1.2896783351898193, 0.12012308835983276, 0.15836048126220703, -0.47504961490631104, 0.2881332039833069, 0.06940055638551712, 0.9001127481460571, -0.2041868418455124, -1.6313310861587524, 1.1428298950195312, 0.6657795906066895, 0.9807119965553284, 0.7866405248641968, -1.6236733198165894, 2.211556911468506, 1.976629376411438, -0.2727157473564148, -0.7592760324478149, -0.6624000072479248, -0.6832334995269775, -0.4535791873931885, 0.9996987581253052, 0.6068860292434692, 0.4745025038719177, 0.9437766671180725, 0.6034359931945801, -2.106691837310791, 1.0094873905181885, 0.3334319591522217, -1.1552499532699585, 0.3423328101634979, 1.1456972360610962, 0.7450716495513916, 0.07469990849494934, 2.470785617828369, -1.6844520568847656, 0.0755573958158493, -0.8256655335426331, -0.7739307284355164, 0.5405124425888062, -0.4783138036727905, -0.704078733921051, -0.7356189489364624, -1.7425682544708252, 0.09030493348836899, -1.2054951190948486, 0.9063469767570496, -1.7103244066238403, 1.3917678594589233, 2.9122581481933594, -0.6143805384635925, -0.7552535533905029, -0.7194985151290894, -0.7057989835739136, 0.5754778385162354, -0.16676995158195496, -1.7076520919799805, 0.22697225213050842, 0.27867770195007324, 0.21714018285274506, -0.5652585029602051, 0.01765880547463894, 0.11410467326641083], [1.5154087543487549, 0.3192194104194641, -0.07480264455080032, -1.078963279724121, -1.4411712884902954, 1.4114117622375488, -0.4379889965057373, 1.5907368659973145, -0.6968638300895691, 1.6697684526443481, 1.2527422904968262, -0.45125913619995117, 1.4075077772140503, 0.8143274188041687, 0.45568081736564636, 0.024360444396734238, -0.04420242831110954, 1.3526897430419922, 1.5289714336395264, 0.4374496340751648, -0.5386880040168762, 2.620389223098755, -1.9292361736297607, -2.414815664291382, 0.7526798844337463, 0.7857122421264648, 0.8817675709724426, -0.22947314381599426, 1.2053122520446777, 1.3492308855056763, 0.7250182032585144, 0.9066879153251648, -0.07346992939710617, 1.0685590505599976, 0.5639780759811401, 0.25220683217048645, -0.14555723965168, -0.500480592250824, -0.7724088430404663, -1.0824049711227417, -0.03580267354846001, -1.2569252252578735, 0.22347940504550934, 0.07871677726507187, 0.00708010932430625, 0.16308631002902985, 0.5278992056846619, -0.07051099091768265, 0.4299173355102539, 0.020855726674199104, -0.6561575531959534, -0.02964097075164318, 0.2406478375196457, -1.3026175498962402, 0.18927356600761414, 0.48959940671920776, -1.5531375408172607, 1.5089032649993896, -0.13675053417682648, -0.6321784257888794, 0.6177763342857361, -0.0558488667011261, -0.7238211035728455, -1.8658150434494019, 0.5537014603614807, 1.7625797986984253, 0.31297144293785095, -0.9140850901603699, 0.2442699670791626, -1.5211416482925415, -0.1976393461227417, -0.6967612504959106, 0.22857271134853363, -0.822930097579956, 0.5975597500801086, 0.4973660111427307, 1.0348328351974487, -1.1503607034683228, 2.597313165664673, 1.5025917291641235, 0.025064710527658463, 1.8230679035186768, 0.36997362971305847, 0.9379059076309204, -1.0128967761993408, 0.4911002218723297, 0.548396110534668, -0.23744039237499237, 0.253487765789032, -0.05830216780304909, 0.7763683199882507, 0.9108672142028809, -0.9412415623664856, -0.010324000380933285, -1.351486086845398, -2.7280352115631104, -0.16394855082035065, -0.6928411722183228, 1.4863581657409668, -1.8700796365737915, -0.20428244769573212, -0.612625777721405, 0.41429227590560913, 0.6657811999320984, 0.3788726329803467, -2.5325145721435547, 0.27978241443634033, -0.32164543867111206, 0.8266626596450806, 0.5632309913635254, -0.6375461220741272, -1.704068660736084, 0.1807529181241989, -0.8338614702224731, 1.396114706993103, -1.4455971717834473, -0.1259499341249466, -2.2952263355255127, -0.1654852330684662, 0.22476741671562195, -0.5008773803710938, 0.1935606300830841, -0.8007928133010864, 0.7563053965568542, 0.12095217406749725, 0.0794520229101181, 1.1790673732757568, 0.2883424162864685], [-1.0103849172592163, 1.441996693611145, -1.6855729818344116, -0.47689178586006165, 0.8293086290359497, 0.6795175075531006, -1.139176845550537, -1.9411641359329224, 1.5408607721328735, -1.5801453590393066, -1.106080412864685, 0.09812891483306885, -1.3233444690704346, -0.04220627620816231, -0.5420350432395935, -0.03718838468194008, -0.9564113020896912, 0.9151192307472229, -0.43564194440841675, 0.06493586301803589, 0.6111485362052917, 0.2929941713809967, -1.173938512802124, -0.6253952980041504, 1.2404553890228271, 0.05883410945534706, -0.14489808678627014, -1.0955654382705688, -2.3417913913726807, 1.0521538257598877, 1.2567579746246338, -0.45788875222206116, 0.6989341378211975, 1.6496293544769287, 0.4687209725379944, 0.11669591814279556, -0.28408893942832947, 0.11210179328918457, -0.6120401620864868, 0.5458862781524658, 0.01811082661151886, 0.7600155472755432, -0.3505917191505432, 1.2752894163131714, -0.774315595626831, -0.43851783871650696, 1.2605557441711426, -2.2421071529388428, 0.6975842118263245, -0.3683820366859436, -0.1257418543100357, -0.005422830116003752, -0.14644740521907806, -0.6096228957176208, -0.2819939851760864, 0.7027990818023682, -0.06861531734466553, -0.3150642216205597, 0.8169461488723755, 1.423809289932251, -0.09777780622243881, 0.4240037202835083, 0.3186601400375366, 1.0262937545776367, 0.7250328063964844, -0.6396105885505676, 0.5649417638778687, -0.5542802214622498, -0.2778129577636719, -0.642840564250946, -0.2768184244632721, 0.8210796117782593, -1.095991611480713, -0.13141393661499023, 0.7970859408378601, -0.6582493782043457, 1.1357768774032593, -0.3334105312824249, 0.3446728587150574, 0.17069052159786224, 0.3317885100841522, 0.8688930869102478, 0.32540062069892883, 0.761480450630188, -2.1747329235076904, 0.6730872392654419, 0.09702306240797043, -0.44184255599975586, 0.48887351155281067, -1.2106573581695557, 0.8328756093978882, -0.8786812424659729, -0.12802371382713318, 0.4571654200553894, 0.2054266482591629, -1.083142638206482, -0.701538622379303, -0.9542118906974792, 0.40140780806541443, -1.121822714805603, -1.8821707963943481, -1.4128273725509644, 0.14368122816085815, 0.6711599230766296, -0.5763000845909119, -1.1035528182983398, -0.5107119083404541, -1.5334711074829102, 1.6477676630020142, 1.929569125175476, 0.6495839953422546, -0.05590282753109932, -0.173019677400589, -1.003310203552246, -0.19324098527431488, 0.36316341161727905, 1.3260024785995483, -0.4158453047275543, 0.5871168971061707, -1.608381986618042, 0.1954469233751297, 1.1695897579193115, -0.7795612812042236, -1.2142012119293213, 0.6782609224319458, -0.49885380268096924, 1.3045600652694702, 0.5915175676345825], [-0.24428583681583405, -1.3994156122207642, -0.21229447424411774, -0.4056183397769928, 0.7297934889793396, -1.201417326927185, -2.050187110900879, 0.5890326499938965, -1.2286670207977295, 1.0547226667404175, 0.42403119802474976, -1.2375701665878296, 0.26945117115974426, 1.0233017206192017, -0.8127988576889038, -0.8942549824714661, -0.8188967704772949, 0.9723465442657471, -1.8316086530685425, 0.537536084651947, -0.1412510871887207, 2.5795655250549316, 1.0986918210983276, 1.1711328029632568, 0.9339225888252258, 0.35498109459877014, -0.1998177021741867, 0.9659168720245361, -1.0602681636810303, 1.4136639833450317, 1.5681334733963013, -1.5184351205825806, 0.5049853920936584, 0.03888707607984543, 1.6081725358963013, 0.1213742047548294, -0.12126680463552475, 0.9371384382247925, -1.288477897644043, 0.7715803980827332, -0.25906267762184143, -0.5806604027748108, -1.4530800580978394, -1.5365211963653564, 0.9598013758659363, 0.8798685669898987, -1.0897077322006226, -0.6468079090118408, 1.5488287210464478, -0.8395880460739136, -0.4385188817977905, -1.7180542945861816, 0.5788845419883728, 1.4737544059753418, -0.5401154160499573, -0.747815728187561, 0.361653596162796, 0.8009774088859558, -0.36481261253356934, 0.16136547923088074, 0.8569849133491516, -1.1616463661193848, -0.6647651195526123, 1.5980992317199707, -2.5084609985351562, 0.27387163043022156, -1.0837268829345703, 0.17106205224990845, -0.8951189517974854, 1.389543890953064, -0.45460665225982666, -0.6050605177879333, 1.3977837562561035, -0.33532923460006714, 1.2977792024612427, -0.3460792005062103, 0.1506214737892151, 0.9173771739006042, 0.45681530237197876, -0.6904842853546143, -2.2989957332611084, 0.009767808951437473, -0.5158407092094421, 0.22206152975559235, -0.36696845293045044, -1.9426337480545044, -0.2092273086309433, 0.7245169281959534, -0.16596387326717377, 0.29224085807800293, -1.9145361185073853, -2.2628231048583984, -1.194940447807312, -1.0120327472686768, -0.7009164690971375, 1.878187894821167, 1.6147533655166626, -0.6354615688323975, 0.513319194316864, 0.7911570072174072, -0.4675188660621643, 0.5058299899101257, -0.3912445604801178, -0.7496197819709778, 0.24260544776916504, -1.371777892112732, -0.4709748923778534, 0.3128156065940857, -0.05844428762793541, 0.06953680515289307, 0.35302841663360596, 2.2149267196655273, 0.985759973526001, -1.3756344318389893, 0.6913616061210632, 0.8413196802139282, -0.4940088987350464, -0.30709004402160645, 1.5968090295791626, -1.8727890253067017, -0.8115977048873901, -0.8091955184936523, 1.149231195449829, -0.5041818022727966, -0.37824881076812744, -0.016944680362939835, -1.2107291221618652, -0.7967907190322876], [-0.7132796049118042, 0.5050526261329651, -0.5382265448570251, -0.16337881982326508, 0.04390009492635727, 0.368216872215271, 0.7990867495536804, -0.12614087760448456, -0.2115650475025177, 1.3222404718399048, -0.18414533138275146, -0.8262753486633301, 0.0395328551530838, -0.3034154176712036, 0.19112460315227509, -0.3493344783782959, 0.17511184513568878, 0.6452386379241943, 0.06973132491111755, -0.40176859498023987, -1.11644446849823, -1.3150144815444946, -2.496927499771118, 0.09145786613225937, 0.532892107963562, -0.9125725626945496, 0.8885149359703064, 0.5200193524360657, 0.31742122769355774, 0.05626463145017624, -0.39549580216407776, 1.52711820602417, -0.06378908455371857, 0.5889162421226501, -1.3372044563293457, 0.7315640449523926, -0.39738842844963074, 1.535064935684204, -0.43030402064323425, 0.7680112719535828, -0.05485450103878975, 0.95048987865448, -0.5377920269966125, -1.568776249885559, 0.7681140303611755, 0.6617013812065125, 0.8425474166870117, -0.21496260166168213, 0.4801144599914551, -0.55108243227005, -0.06454722583293915, -2.0603694915771484, -1.6050097942352295, 1.893703579902649, 0.4578307867050171, 1.9694758653640747, -1.5677554607391357, 0.5187757611274719, 1.6010618209838867, 0.3447434902191162, -0.4004327356815338, -1.159131646156311, 1.042615294456482, -0.2842102646827698, -1.5165022611618042, -0.05490608885884285, 0.18580852448940277, 0.35219788551330566, -1.7670814990997314, -1.0235910415649414, 0.02623808942735195, 0.9261633157730103, -1.1892178058624268, 1.7582789659500122, 0.034646596759557724, -0.6002342104911804, -1.000632882118225, -0.21671149134635925, 0.3608051538467407, 0.21232351660728455, -1.0577701330184937, -0.4931659400463104, 0.07340999692678452, 1.7543312311172485, -0.4733564853668213, -1.732055902481079, 1.1183918714523315, 1.8658190965652466, -0.34391704201698303, -0.7989981174468994, 0.15837322175502777, -0.08146501332521439, -0.38783735036849976, -1.497812271118164, -1.039544939994812, 1.1738404035568237, -0.15289415419101715, -0.572913646697998, 0.9845051765441895, 1.3033171892166138, -1.4096338748931885, 0.7086793184280396, -0.690085232257843, 0.38710692524909973, -1.4631402492523193, -0.4258081912994385, 0.8984795808792114, -0.4292192757129669, -0.21201223134994507, 0.5039932727813721, 1.1385619640350342, -0.6150403618812561, -0.28459277749061584, 0.5206075310707092, -0.3196326792240143, 0.040705688297748566, -0.9661145806312561, -1.117978572845459, 1.1328492164611816, 0.3839842677116394, -0.07267850637435913, -1.035636067390442, 0.4216795563697815, -0.21814721822738647, -1.430862545967102, -0.8272383809089661, -0.8192533254623413, 0.17785508930683136], [-0.9486106038093567, -0.2989146113395691, -0.1518661230802536, -0.4844020903110504, -0.6231810450553894, -0.8398474454879761, 0.11700178682804108, 0.13197766244411469, -1.2096816301345825, 0.29927849769592285, -0.6855964660644531, -0.1524457186460495, 2.1678011417388916, 1.1215875148773193, 0.8056054711341858, -0.6217777729034424, 0.1469760537147522, -0.07127255946397781, -0.3980462849140167, -0.45641404390335083, -0.651441752910614, 1.1209262609481812, 0.8992622494697571, 0.5138495564460754, 0.46182388067245483, -0.9039218425750732, 0.6714590787887573, -1.517673134803772, 0.7584510445594788, -1.4741204977035522, 0.05581391602754593, -0.8430885672569275, -0.13777028024196625, 1.0785295963287354, 0.2896682024002075, -1.211863398551941, 1.8629217147827148, -0.29287317395210266, 1.2215300798416138, -1.1994078159332275, -0.19910559058189392, 0.8354569673538208, -0.12647688388824463, 1.6511237621307373, 2.4890286922454834, 0.43650341033935547, -0.6250463724136353, -0.5730199813842773, -0.33255094289779663, 0.7357404828071594, -0.8318642973899841, -0.4889245927333832, 0.28155601024627686, -0.34906917810440063, -0.14997483789920807, 0.027362024411559105, -0.8437819480895996, 1.1413112878799438, 0.027860749512910843, -1.3435171842575073, 0.5666773319244385, -0.5648674368858337, -0.1408453732728958, 0.2557504177093506, 0.25680986046791077, 0.5590620636940002, 0.1988188624382019, 0.15141203999519348, 0.3922370970249176, -0.05992758646607399, -1.52760910987854, -0.05692819505929947, -2.2692551612854004, -0.38624557852745056, -0.20212583243846893, 0.39677688479423523, 1.7714730501174927, 0.46974098682403564, -1.0292463302612305, 1.1397502422332764, -0.8403555750846863, 0.5543559193611145, 1.0226446390151978, 0.38989171385765076, 0.46037358045578003, 0.002408513566479087, 0.17795543372631073, 1.2207111120224, 0.5170705318450928, 1.049392580986023, 1.1455751657485962, 0.06634336709976196, 0.5145936012268066, 0.6610761284828186, -0.44939979910850525, 0.46738144755363464, 1.1480597257614136, 1.2484245300292969, -0.11754801124334335, 0.04711485281586647, -0.7185596227645874, 0.23783724009990692, -1.0914852619171143, 0.31372833251953125, 0.2007957398891449, -2.208019495010376, -0.6491191983222961, -1.524577260017395, -1.5779573917388916, -1.064101219177246, -0.6916552782058716, 0.5990832448005676, -0.8223875164985657, -0.12319561839103699, -0.8879932165145874, -1.003321647644043, 2.1006035804748535, -1.1833738088607788, 0.7681323289871216, -0.27292224764823914, -0.6879467368125916, -0.7823575139045715, 1.1985524892807007, 0.33782926201820374, -1.0903927087783813, -0.8668261766433716, -1.044975757598877, 0.5199653506278992]], [[0.8898592591285706, 0.11592521518468857, -0.24902048707008362, -0.06277771294116974, -0.684093713760376, -0.6251939535140991, 0.17621320486068726, -0.8610816597938538, -1.2999659776687622, -1.0224666595458984, 0.2106967270374298, -1.0782761573791504, -0.11196643114089966, -1.0059738159179688, -1.485027551651001, -0.6205669045448303, 1.9722548723220825, -0.016081677749753, 0.7846934795379639, -1.5698330402374268, -1.090921401977539, 0.5852490067481995, 1.416765570640564, -2.0972416400909424, -1.0948435068130493, -0.3932490944862366, 2.050558090209961, 0.8721380829811096, -1.749694585800171, 0.5432680249214172, 0.3710459768772125, 0.6168151497840881, -0.35596367716789246, 1.5637226104736328, 0.8059526681900024, -2.341721534729004, 1.206620454788208, 1.002734661102295, -1.6147710084915161, 0.12540803849697113, 0.7984763979911804, 0.8419528603553772, 0.12044163793325424, 1.1157407760620117, 0.35930904746055603, 1.8799363374710083, -0.7906107306480408, 0.5284229516983032, -1.2294597625732422, 0.8622264266014099, -0.34023115038871765, 1.1307669878005981, -0.6745755076408386, 0.35861456394195557, -0.28630420565605164, -1.2165552377700806, 0.2011740505695343, 0.8955813050270081, 0.43421056866645813, 0.8280060291290283, -1.4350199699401855, -0.447116494178772, -0.23130619525909424, 0.4717118442058563, 1.406071662902832, 1.0862864255905151, -1.3265670537948608, -3.154057264328003, -1.7336610555648804, -1.9629288911819458, 0.7089514136314392, 0.5027465224266052, 1.636680245399475, 0.13595977425575256, -2.06424880027771, 1.4409942626953125, 0.8518317341804504, -0.7195355892181396, 0.6629471778869629, -0.5785166025161743, -0.35686907172203064, -0.3552856147289276, 1.4720507860183716, -1.2931251525878906, 1.4394958019256592, 0.581195592880249, 1.1896469593048096, -1.851188063621521, -0.16912133991718292, 0.49017333984375, 0.3303217589855194, 0.23597054183483124, -1.2094018459320068, 0.10918310284614563, 0.3905787467956543, 0.005636183079332113, -0.2934451699256897, -0.4288868308067322, -0.294382780790329, -0.4996834099292755, 0.9555712342262268, -1.3768614530563354, -0.007832442410290241, -0.5802300572395325, 0.7616788148880005, 0.5213080644607544, -0.1362404078245163, 2.0970938205718994, 0.14996542036533356, -0.6773099303245544, 1.520397424697876, -0.6400196552276611, -1.269835352897644, 0.3574499189853668, 1.6528595685958862, 0.4218524992465973, -0.17785270512104034, 0.7044246792793274, 1.0093036890029907, 1.075732946395874, 0.12431564927101135, 0.03455961495637894, -0.10925588756799698, -1.5102113485336304, -0.8276099562644958, 0.6258722543716431, -1.216320276260376, -0.694000244140625], [-0.4056631028652191, 1.1529245376586914, -0.37117528915405273, -1.3759697675704956, 0.13895642757415771, -1.2484484910964966, -0.10280536860227585, -0.057424310594797134, -0.23811687529087067, 0.9698678851127625, 1.6468995809555054, -0.7327072024345398, -0.06900745630264282, -0.4261666238307953, 0.04247170314192772, -0.826161801815033, -0.16017599403858185, 2.7740390300750732, -0.3413236141204834, -0.503217875957489, -1.7071950435638428, -0.728168249130249, 0.9056350588798523, 1.3901845216751099, -0.6708559393882751, -0.42187872529029846, -1.644471526145935, 0.6888418793678284, 1.8614482879638672, 1.0246325731277466, -0.7260721921920776, -0.2816004455089569, 1.9049819707870483, -0.12729208171367645, -1.0479556322097778, -0.09746651351451874, 0.18606777489185333, 0.850608766078949, -0.4353177547454834, -0.733173131942749, -0.41898611187934875, -0.7555115818977356, 1.3014841079711914, 0.7699338793754578, -0.4797736406326294, 1.9086402654647827, 0.17900162935256958, -0.7208360433578491, -1.1130354404449463, -0.7479997277259827, -0.6790528893470764, 1.947982668876648, 0.26179268956184387, 1.079663872718811, -1.134002447128296, 1.8128514289855957, 0.8936378955841064, -0.5232741236686707, -0.022859910503029823, 0.6304248571395874, 0.5254692435264587, 0.2967170178890228, -0.9463772773742676, -0.6555909514427185, -0.5814933776855469, 0.3245108425617218, -1.3059359788894653, 0.0669291689991951, 0.2507278025150299, -0.8707872033119202, 0.2751263380050659, 1.1851588487625122, -1.2233421802520752, -0.2619841992855072, -0.23479127883911133, 1.7341872453689575, -0.6926006078720093, 0.13236358761787415, 0.751370370388031, 0.045162566006183624, -0.22087037563323975, -1.2140322923660278, 1.5323066711425781, -0.9100985527038574, -0.6908209919929504, -0.8079398274421692, -0.20826996862888336, 0.35033902525901794, -1.2531050443649292, -0.6273778676986694, 0.4271191656589508, -0.15737107396125793, 0.016367273405194283, 1.2037500143051147, 0.9844134449958801, -2.0856916904449463, 1.0068279504776, 1.1788277626037598, 0.9783933162689209, 1.5539188385009766, -0.26878225803375244, 1.0311187505722046, -0.42806291580200195, 1.2593172788619995, 1.6822631359100342, 1.5106781721115112, 0.41297608613967896, 2.6024856567382812, -0.9864934086799622, 0.9153038859367371, -1.0550446510314941, 0.513306200504303, 1.6452679634094238, -0.43560466170310974, -0.7381916642189026, 2.906200647354126, 0.7925059199333191, -1.8445155620574951, 0.37241485714912415, -0.8578010201454163, -1.585360050201416, 0.8583605289459229, 0.32467278838157654, 0.11999693512916565, 1.8727024793624878, -1.6691805124282837, -1.0710887908935547, 1.2327191829681396], [0.09501849859952927, 0.4488329291343689, 2.196078300476074, 0.5531048774719238, 1.3939343690872192, -0.2840711772441864, 0.15046019852161407, -0.6299558877944946, 1.623311996459961, -0.19638149440288544, 0.8630428314208984, 0.6270314455032349, 1.1834441423416138, -1.5202778577804565, -0.7707908749580383, -0.19575391709804535, -0.8355717658996582, 0.16375823318958282, -0.17249098420143127, 0.5919408202171326, -0.7504158020019531, 1.442728877067566, -0.05584347993135452, -0.42351633310317993, -0.8275933265686035, 0.4197070896625519, -1.8737072944641113, -0.8317362070083618, 0.14731024205684662, 0.441739946603775, 0.016958650201559067, 0.4327848553657532, -0.7865591049194336, 2.1532583236694336, 0.5299566388130188, 0.7641536593437195, 1.678269863128662, -0.16982047259807587, 0.511989176273346, -0.5054499506950378, 0.2256269007921219, 0.0997229740023613, 0.11747045814990997, -1.1717793941497803, 0.280122846364975, -0.17907072603702545, 0.7553830146789551, -1.0953214168548584, 0.3638654947280884, 1.396120548248291, -0.43627843260765076, 0.7454147934913635, -0.5308341383934021, -0.8840515613555908, 0.3649204969406128, 0.966407299041748, -1.2010047435760498, -0.8434270024299622, 0.450111985206604, 0.028792478144168854, -0.213418111205101, -1.1923043727874756, 0.0244553554803133, -1.8060060739517212, 0.6779887676239014, 0.5498155355453491, 0.5770070552825928, 1.4885603189468384, 0.2608475983142853, 0.5957857966423035, -1.2183685302734375, -0.6395339965820312, 0.7507774233818054, -0.11532702296972275, 0.3013802468776703, -0.3375069200992584, -0.8279799818992615, -0.4534488022327423, 1.072898268699646, 0.9111570715904236, 1.0543593168258667, -1.60887610912323, -0.9144587516784668, 0.057383447885513306, 0.5139980316162109, 0.47818952798843384, -1.6568171977996826, -0.12997369468212128, 0.008968422189354897, -0.144250750541687, 0.9666402339935303, 0.8788735866546631, -1.3698773384094238, 0.14148841798305511, -0.13346119225025177, -2.456557512283325, -0.48842352628707886, -2.5854666233062744, -0.415323406457901, 0.5021994709968567, 0.2326478511095047, 1.4780323505401611, -0.3830656111240387, -0.3003598153591156, 0.47294557094573975, -0.35583704710006714, 0.24002134799957275, -0.041277844458818436, -0.45961859822273254, 0.4651612639427185, -0.7381296753883362, -1.1164017915725708, 1.1792926788330078, 0.6849815845489502, -1.4910520315170288, 0.4192158579826355, -1.3443372249603271, -0.6419165134429932, -0.33448857069015503, -1.388564944267273, 1.6073980331420898, -0.9527348875999451, 0.5582260489463806, 1.730054497718811, 1.2917819023132324, -1.3336361646652222, 0.10626357048749924, 0.10055246949195862], [0.2972176969051361, -1.5845082998275757, -1.3215138912200928, 1.7778961658477783, 1.4868749380111694, 0.979780912399292, -1.1452423334121704, 1.2355984449386597, -0.012560883536934853, -0.00689340615645051, -0.3429465591907501, -0.734725296497345, 1.0529065132141113, 0.10812105238437653, 0.7040907144546509, -1.576737642288208, 0.9225882887840271, -1.2075917720794678, -1.4433313608169556, -1.0981007814407349, 1.4188121557235718, -0.25178298354148865, 0.149018794298172, -0.31763511896133423, 0.2014215588569641, 0.06592299044132233, -0.14710773527622223, -0.1435307115316391, 0.3207046091556549, 0.03918763995170593, -0.8275431990623474, -1.3410333395004272, -1.88341224193573, 0.746933102607727, 0.6921640038490295, 1.374396800994873, 0.14381250739097595, -2.0952582359313965, -0.08100611716508865, -0.9784952998161316, -0.8286218047142029, -0.48762866854667664, 0.7640124559402466, -0.4085543155670166, -0.7247747182846069, 0.2756480574607849, 0.34217172861099243, 0.6160219311714172, 1.288006067276001, 0.8994787931442261, -0.417452335357666, -1.1394401788711548, -1.85562264919281, 0.7494668364524841, 0.7390293478965759, 0.4757644832134247, -0.9850291013717651, -1.2285776138305664, 1.1607223749160767, 0.21169711649417877, -1.191197395324707, -1.4740605354309082, -0.20477402210235596, -0.8126850724220276, 0.6823421120643616, -0.7440894842147827, -0.926405668258667, 0.12885507941246033, 0.052675653249025345, 1.784440040588379, -0.5636063814163208, 0.12465984374284744, 1.1233028173446655, 0.9666388034820557, -1.126112461090088, 2.315310478210449, -0.26657018065452576, -0.6741905212402344, -1.0280027389526367, -1.1585643291473389, -0.4991324543952942, 0.8795225620269775, 0.8128688335418701, -0.6328669190406799, -0.8330298066139221, -1.3033666610717773, 0.5359094142913818, -0.27792301774024963, -0.3650801479816437, 0.2866334021091461, 0.12522117793560028, -0.6211904287338257, -1.2929903268814087, -0.6683846712112427, -0.30823594331741333, 1.7933251857757568, 1.0306973457336426, -0.25718337297439575, -0.6664199829101562, -0.5740509033203125, 0.6417351961135864, 0.48863816261291504, 2.2371840476989746, -1.5849180221557617, -0.8153302669525146, 0.8464760184288025, -0.9026055932044983, 0.4773930311203003, 0.3023938834667206, -0.8794963359832764, 0.6466519236564636, 0.6597427129745483, 0.5982027649879456, 0.8808813691139221, -0.07508319616317749, -0.1213671937584877, 0.134023517370224, -0.6052871346473694, -1.1774184703826904, 0.8486011028289795, 0.07835635542869568, 0.11665024608373642, 0.6663748621940613, -1.4643330574035645, -0.06747083365917206, -0.24336545169353485, 1.3739640712738037, -0.053228262811899185], [0.7837756872177124, 0.4026351273059845, -1.2178583145141602, 0.3519406020641327, -0.2904053032398224, -0.43548211455345154, -0.5737076997756958, -0.04930824041366577, -0.6149832606315613, -1.6326756477355957, 0.40254536271095276, 0.6287960410118103, 0.4346712827682495, -0.17980524897575378, -0.15989205241203308, 2.357734441757202, 0.19000397622585297, 1.5178436040878296, -0.3855418860912323, 0.9386816620826721, -0.3909604847431183, 1.5398725271224976, -1.2812399864196777, -0.4407518208026886, 0.06240268796682358, 0.24858425557613373, -0.05878029018640518, 0.07359621673822403, 0.8059085607528687, 0.18593347072601318, -0.8917683959007263, -0.05371556058526039, -0.10850012302398682, 0.13526394963264465, -1.3357996940612793, -0.46571972966194153, -1.9697028398513794, -0.09481611102819443, -0.906337559223175, 1.7388592958450317, -0.49467557668685913, 1.118815541267395, 1.4516141414642334, -0.3527531623840332, -0.9227821826934814, 0.23900291323661804, -0.8130252361297607, -0.32116782665252686, 0.010796484537422657, -0.7237520813941956, -0.9040420651435852, 2.4253933429718018, 0.7131445407867432, -0.3666638731956482, -0.15939436852931976, 0.4626694321632385, 0.34626415371894836, -2.079375982284546, -0.21949368715286255, -0.23942306637763977, -0.5587986707687378, -1.4526923894882202, -1.7387512922286987, 0.03334061801433563, -0.3630983233451843, -2.1884002685546875, -1.15558922290802, -0.4499199390411377, 0.08311102539300919, -0.10316554456949234, 0.3726934492588043, 1.2674731016159058, 1.2204841375350952, 0.16564401984214783, -1.6028310060501099, 0.32460665702819824, 1.7903625965118408, -0.9270439743995667, -0.5654392838478088, -1.6500447988510132, 1.0725938081741333, 0.5575348138809204, -0.14298270642757416, 0.41721606254577637, -0.26523345708847046, -0.9631928205490112, 0.2254214584827423, 0.8217836022377014, 0.6423158049583435, 0.4494193494319916, -1.4600328207015991, -0.47888579964637756, 0.10504209995269775, -1.8880685567855835, 1.2303191423416138, -1.6939536333084106, 0.6148490309715271, 0.25341811776161194, 1.615917444229126, 0.786365807056427, 0.8449735045433044, 0.879097580909729, 0.9411230087280273, 1.6877127885818481, 1.1214346885681152, 1.0783286094665527, -1.0314987897872925, 0.44835713505744934, -0.10632748901844025, 0.7507631182670593, 2.1453983783721924, 0.17471319437026978, 0.2252795696258545, 1.0970951318740845, 1.0567442178726196, -0.8108519315719604, -1.192962646484375, -0.27320700883865356, 0.3815757930278778, 0.2717765271663666, -0.1117224395275116, 0.3166202902793884, 0.44839048385620117, 0.48136255145072937, 1.6449333429336548, -0.8550698757171631, 0.4474802017211914, -0.9902780652046204], [-1.7953815460205078, -1.0246762037277222, -0.151696115732193, -0.8746567964553833, -0.77545565366745, 0.800879955291748, -0.23920771479606628, -0.03629259765148163, 0.5905879735946655, -1.1802512407302856, 0.5306807160377502, -0.6596192717552185, -1.2258336544036865, -0.7482863068580627, -0.6777781248092651, -0.0828525647521019, 1.203485131263733, -0.1905028522014618, 0.6416381597518921, 0.3964647948741913, 1.6305245161056519, -0.7642828226089478, 0.15577419102191925, 0.11892347037792206, -0.4150698781013489, 0.17659427225589752, 1.200825810432434, 1.719704270362854, 0.46259909868240356, -0.9872925877571106, 2.7457971572875977, -1.1998651027679443, 0.47042420506477356, -0.3807279169559479, 0.398173063993454, -1.0397188663482666, 0.7483489513397217, 0.4160259962081909, 0.3710422217845917, 0.7004520893096924, 1.4584598541259766, -1.0773001909255981, 0.22439783811569214, -0.5544552206993103, 0.2564942538738251, -0.7231090664863586, -1.58539879322052, -0.7768790125846863, -0.8597514629364014, 0.1478801816701889, -0.47058361768722534, 0.17150014638900757, -0.037718258798122406, 0.3365600109100342, -0.9233350157737732, -0.8244624733924866, 1.0514785051345825, -1.6447045803070068, 0.2968863546848297, 0.4640762209892273, -0.17359322309494019, -0.5994976162910461, 0.6960211992263794, -0.7698403000831604, 0.8737283945083618, 0.2681262493133545, -0.646574079990387, 0.13877493143081665, 0.583413302898407, 0.3129281997680664, 1.9856443405151367, -1.6780989170074463, -0.8836488127708435, 0.012694176286458969, 0.005348179489374161, -0.5974039435386658, -1.4382978677749634, -0.7892624139785767, 0.3700694739818573, 1.4562053680419922, -0.1632477194070816, -0.2161174714565277, -0.9732146859169006, 1.3949799537658691, -2.0330843925476074, -0.5838083624839783, 3.442131996154785, -1.686100721359253, 0.05310773476958275, 1.258931040763855, 0.2827102541923523, -1.1693295240402222, 0.8545046448707581, 0.01004084199666977, 0.8836480379104614, -0.3257581889629364, 1.1181586980819702, -0.16642868518829346, -0.18412485718727112, -2.455518960952759, 0.6505336165428162, 0.0730457603931427, -0.36375290155410767, -0.6018885970115662, 0.14853601157665253, -0.6653783321380615, 0.20115818083286285, -0.5269179940223694, 1.418228268623352, -0.0541754886507988, 1.8128641843795776, 1.0463001728057861, 0.48562535643577576, -0.6158432364463806, -0.4403194189071655, 0.14740756154060364, 1.3749788999557495, -2.11977481842041, -0.9713029265403748, 1.3793565034866333, 0.04717756062746048, -0.1976793110370636, -0.5293439030647278, 0.75202476978302, -0.19922861456871033, 0.26136496663093567, -0.3805093765258789, 1.0542066097259521], [1.8064541816711426, -0.7752301096916199, 0.42093625664711, 0.22551700472831726, 0.023694079369306564, -0.549004316329956, 1.4654529094696045, -0.033805422484874725, 1.090414047241211, 0.4483012855052948, -0.6149506568908691, -1.6979830265045166, 1.3962148427963257, -1.4894565343856812, 0.6737571358680725, -0.7952552437782288, 0.21561025083065033, 0.4319184720516205, -1.2246887683868408, -0.2498553842306137, 0.3626358211040497, 0.8533279895782471, -1.6234986782073975, 0.4976451098918915, -0.20275361835956573, -2.802215337753296, -0.6581801772117615, 0.9168344736099243, 0.8985516428947449, -1.7269530296325684, 0.45680785179138184, 0.7325377464294434, 0.5402076840400696, 0.3676682412624359, -1.264276385307312, -0.21957670152187347, -0.7011529207229614, 0.13969823718070984, -1.5764131546020508, 0.22000885009765625, -0.11711760610342026, 0.7504493594169617, 1.1158174276351929, -1.8901076316833496, -1.064773440361023, 0.2595363259315491, 0.26205524802207947, 0.3640173077583313, 0.33069971203804016, 1.5776054859161377, 0.586466372013092, 0.895951509475708, -0.9602078199386597, 0.6215816140174866, 0.962177574634552, 1.7234004735946655, 0.3702818751335144, -2.135040521621704, -0.6292930841445923, -0.04546252265572548, 0.3967784345149994, 0.8145235776901245, -0.991901695728302, 0.15312649309635162, -2.234929084777832, 0.5754552483558655, 2.3428776264190674, 0.12716799974441528, -0.14274534583091736, 1.2128214836120605, -0.6788432598114014, 1.3479501008987427, 0.21754567325115204, -1.090696096420288, -2.8123509883880615, -2.220968723297119, -0.6432248950004578, 1.1689724922180176, 1.8613241910934448, 0.13402239978313446, 0.6311283707618713, 0.6726282835006714, -0.4748622477054596, 0.3864274322986603, -0.9653041362762451, -1.138265609741211, -1.0298715829849243, -0.6579838991165161, -0.13217705488204956, 1.509233832359314, -1.0720199346542358, 1.168326497077942, -1.7065755128860474, -0.4918966591358185, 0.5269500613212585, 0.09523113071918488, -0.30395445227622986, -0.2062627673149109, 0.09207438677549362, 0.8085255026817322, 0.43762946128845215, 0.858780562877655, 0.42590799927711487, 0.5505397915840149, -0.4651493430137634, -1.0588675737380981, -1.4795527458190918, 0.6267341375350952, 1.0752887725830078, -0.21188722550868988, 0.2147093117237091, -0.4072709381580353, -0.9202497005462646, -2.078341007232666, -0.08013182878494263, -0.9258205890655518, 0.6542883515357971, -0.9719453454017639, -0.09549248963594437, -1.0241886377334595, -0.2807232439517975, -1.9854226112365723, -0.517096221446991, -1.0222951173782349, -0.49346423149108887, 0.3583884835243225, -0.09699223935604095, -0.5190856456756592], [0.21844112873077393, -0.8087600469589233, 0.6323029398918152, -2.4301915168762207, -1.1237993240356445, -1.165894627571106, -1.1439487934112549, 0.7759395837783813, -2.19856333732605, -1.6580727100372314, 0.4852290749549866, -0.7765201926231384, 0.7064367532730103, 0.6114760637283325, 1.9971184730529785, -0.10050065815448761, 0.8191957473754883, -0.16712528467178345, -0.617238461971283, -0.8160079717636108, 0.9343365430831909, 1.1978044509887695, 1.394138216972351, 0.5995955467224121, -0.6894697546958923, 0.7514786720275879, -1.6545850038528442, 0.025250956416130066, -0.9396792054176331, -0.81259685754776, 1.6781842708587646, -2.0448925495147705, 2.3322789669036865, -0.7063444256782532, -1.1771302223205566, 0.3697565793991089, -0.8526726961135864, -1.8860677480697632, -0.7762948274612427, 0.3680468797683716, -1.1254469156265259, 0.32295069098472595, 0.29777824878692627, 0.7853938937187195, 0.4548477232456207, -0.829376757144928, -0.4352428913116455, -1.2094907760620117, 0.3389360308647156, 0.8776158690452576, -0.5066264271736145, 0.9629572629928589, 0.41838544607162476, 0.745306134223938, -2.278247833251953, -1.4358819723129272, -0.04544772580265999, 1.0066123008728027, -0.21178045868873596, -0.6727462410926819, 0.21710895001888275, 0.1884792596101761, -0.23034213483333588, 1.0097169876098633, 2.185431957244873, 0.361245334148407, -1.830413579940796, 1.018941879272461, -0.3439845144748688, 1.0126442909240723, 1.4818572998046875, -0.3959191143512726, 0.741183876991272, -1.1573439836502075, 0.18661074340343475, -0.1383439004421234, -0.6147711873054504, -0.4535585045814514, 0.6593882441520691, -1.1655985116958618, 1.3395274877548218, 0.9614936709403992, -0.07982521504163742, -0.7652336955070496, -0.04608309641480446, -0.03049401007592678, -1.2683658599853516, -0.2720962464809418, 0.2800336480140686, 0.5091968178749084, 0.4560443162918091, 0.2517731189727783, 1.9326554536819458, 0.9803484082221985, -1.482728123664856, 2.4018707275390625, -0.452515184879303, 0.7121784687042236, 1.2304328680038452, -0.5126878619194031, 0.0639573261141777, -0.5579178333282471, -0.267506867647171, 1.0435450077056885, 0.18844155967235565, -0.8320562839508057, 0.0717969760298729, 0.10264062881469727, 1.6719589233398438, -1.126114845275879, -1.556438684463501, 1.2770715951919556, -3.549931764602661, -0.42367270588874817, -0.9163480401039124, 0.9565932154655457, -2.1387393474578857, -1.3416777849197388, -1.354066014289856, -0.6043115854263306, 0.27408263087272644, 0.019362235441803932, 1.181049108505249, -0.43829289078712463, 1.0389409065246582, -0.2692163288593292, 0.20061451196670532, 0.5524035692214966], [0.5913585424423218, -0.34721627831459045, -1.4290621280670166, 0.6877449154853821, 0.6286723613739014, -0.9549245834350586, 0.011646859347820282, 0.03882549703121185, 0.4091457426548004, 0.9135241508483887, -1.0779423713684082, -2.3728206157684326, 0.20652557909488678, 1.6074398756027222, 0.08538378775119781, 0.41007813811302185, 1.3949085474014282, -0.10199613869190216, -0.537472665309906, 1.2985668182373047, -0.2440386861562729, 1.6293842792510986, 1.0518584251403809, 0.08563588559627533, 1.3079639673233032, -2.2580649852752686, 0.8387994170188904, 0.6021130681037903, 0.1350405514240265, -0.9260413646697998, -0.010380326770246029, -0.000581785396207124, -1.5191068649291992, -2.147512674331665, -0.7401267886161804, 1.2426317930221558, -0.6004310846328735, 0.5144372582435608, -1.4158549308776855, -1.0728968381881714, 0.014250160194933414, -1.750279188156128, -0.12413884699344635, 1.1027401685714722, 0.19592367112636566, -0.3762477934360504, 0.032750051468610764, 0.8369601964950562, 0.4235393702983856, 0.784629762172699, 1.2504130601882935, -0.6250334978103638, -0.4824952185153961, 0.5745760798454285, 0.6237673759460449, -1.2423850297927856, -1.6541489362716675, 0.08345140516757965, 1.0206092596054077, -0.26365023851394653, -1.7266765832901, 0.8247437477111816, 0.7009363174438477, -0.9408755302429199, 0.5383453369140625, -0.7826209664344788, 0.747759222984314, 0.229447141289711, -0.5848471522331238, -0.3839030861854553, 0.7057527899742126, 0.5350490808486938, -1.4920132160186768, 0.09147225320339203, 1.4577425718307495, 0.24057237803936005, -0.7669947743415833, -1.9609445333480835, -0.28512269258499146, -0.16356636583805084, -0.1318148374557495, -0.040768347680568695, 0.36377617716789246, -0.7802062034606934, -1.0046077966690063, 0.3790433704853058, 0.30810704827308655, 1.3891676664352417, -0.9171674847602844, 0.41122230887413025, 0.169700488448143, 1.2930160760879517, -0.13287082314491272, -1.97343111038208, -0.38949012756347656, 0.44606077671051025, -0.45184454321861267, -1.3831546306610107, -0.7835755944252014, 0.6429517269134521, 0.6598156094551086, -0.2092674970626831, 1.0739201307296753, 1.3150455951690674, 1.233573079109192, -0.9879544377326965, -1.2536466121673584, -1.3340774774551392, -1.092559576034546, 0.06474325060844421, 1.0837318897247314, -0.0037572190631181, -0.9231420159339905, 2.670734405517578, 1.509256362915039, -0.8883333206176758, 0.032642994076013565, 0.49729448556900024, -0.8201123476028442, 0.6500799655914307, -1.0598410367965698, 1.9644333124160767, 0.19015711545944214, 1.5084068775177002, 0.2518922984600067, -2.141955852508545, 1.8030871152877808, -0.5331553220748901], [0.24778491258621216, 1.448352336883545, 0.07658503204584122, 0.4390040636062622, -0.687034547328949, 0.37000373005867004, 1.2065528631210327, 0.6936435103416443, -0.7273667454719543, 0.6214648485183716, 0.19116084277629852, 1.555356740951538, 0.36490070819854736, -0.7937716841697693, 1.3691297769546509, 1.0642821788787842, -0.6827324032783508, -0.7106914520263672, 0.07221357524394989, 0.8681036233901978, 0.4246540069580078, 0.3730054199695587, -0.9938676953315735, -1.306542992591858, 0.30115649104118347, 0.9184621572494507, -1.3308416604995728, -0.8045408129692078, -0.06103784218430519, 0.4113718271255493, 1.341027855873108, 0.46805840730667114, 0.7123719453811646, 0.312493234872818, -0.028556380420923233, 0.7563540935516357, -0.6236249208450317, -0.020687727257609367, 0.4250575602054596, 0.5129119157791138, 0.06353789567947388, -0.9747154712677002, -0.32149970531463623, 1.223728895187378, 0.34230202436447144, -0.9765229821205139, -0.4942716062068939, -1.034490704536438, 0.11710795015096664, -0.25226160883903503, -1.3889939785003662, 0.2666930556297302, 1.156688928604126, -0.9145116209983826, -1.3239436149597168, -1.2607976198196411, -0.18615202605724335, 0.5878451466560364, 0.4168824553489685, -0.9179140329360962, 1.2820720672607422, -0.002221380826085806, -1.271609902381897, -1.1449389457702637, 0.008188314735889435, 0.9074916243553162, 0.18345317244529724, 0.050957243889570236, 1.1770120859146118, 0.3775657117366791, -0.43654367327690125, 1.191926121711731, 1.112828016281128, 0.5444913506507874, -0.7339054942131042, -1.2982150316238403, 2.0694754123687744, -0.16707782447338104, 0.6585775017738342, 0.3277367055416107, -1.2659882307052612, -1.381279706954956, -1.340161919593811, -1.3101338148117065, 1.7391542196273804, 0.11252778023481369, 0.7930861711502075, -0.5184547901153564, -0.3398778736591339, -0.019898049533367157, -0.10935015231370926, 0.9599418640136719, 2.1246659755706787, -0.4407298266887665, -1.5903027057647705, -1.0242401361465454, -0.3570990264415741, -0.8239063024520874, -0.6503742933273315, -0.41828927397727966, 0.34857720136642456, -0.5122992992401123, 1.2979401350021362, -0.8017496466636658, -0.24940061569213867, 1.649324893951416, 0.635148823261261, 0.02882741577923298, 0.37425824999809265, 0.20244824886322021, 1.3161793947219849, 0.3308844566345215, 0.48623713850975037, -0.46792975068092346, -1.7492164373397827, 1.6154834032058716, 0.996669352054596, -1.4630333185195923, 0.9560880064964294, 0.31317004561424255, 1.942195177078247, -0.5044452548027039, -1.0658499002456665, -0.010362474247813225, 1.1936506032943726, 1.3684074878692627, -2.111884593963623, 1.6247638463974], [0.3600609600543976, -1.4320383071899414, 1.006332278251648, -0.9678976535797119, -0.06644529849290848, 0.45013463497161865, 1.2213318347930908, 0.6772769093513489, -0.08395291864871979, -0.26809385418891907, -0.12233539670705795, -0.7361699342727661, 0.6580674052238464, 0.8723688125610352, 0.8841757774353027, 0.1849411278963089, -0.41849014163017273, 1.01029372215271, -0.2653072476387024, -1.486132264137268, 0.8893150687217712, 0.4458526372909546, 0.13066889345645905, -0.23619522154331207, 0.10865695029497147, -0.4833810329437256, 1.65391206741333, -0.8553300499916077, -0.09975258260965347, -1.3841261863708496, 1.0024397373199463, -0.4630677103996277, -0.043125372380018234, 0.5982807278633118, -0.37273192405700684, 1.457540512084961, -0.13699647784233093, -0.7974557876586914, 0.1819283813238144, -0.6346158385276794, 0.712198793888092, 0.25926917791366577, 1.0101935863494873, -0.3027663826942444, -0.8386611938476562, -0.05619243159890175, 0.7027772665023804, -0.3745935559272766, 0.27032557129859924, -0.6023781299591064, -0.5640495419502258, -1.1936732530593872, 1.1907373666763306, 2.1249735355377197, 0.7633127570152283, -0.3448181748390198, -0.6780232787132263, 0.30544063448905945, -0.4704538583755493, -0.41678479313850403, -0.12765228748321533, 0.2942340075969696, -0.2907187044620514, 0.15957088768482208, -0.9072962403297424, 0.45729050040245056, -0.07010750472545624, -0.24956446886062622, -0.5942109823226929, 0.8179271221160889, 0.2302994281053543, -0.15929877758026123, 0.15840284526348114, -1.215207576751709, -0.14600692689418793, 1.6685750484466553, -0.43448954820632935, 0.3283400237560272, -0.003996139392256737, 0.4176047742366791, 0.37268489599227905, -0.8971890807151794, 0.18988023698329926, 0.09922464936971664, 0.7026371359825134, -0.9530865550041199, 1.2576149702072144, -0.7775869965553284, 0.06522638350725174, -0.21680158376693726, -1.7103925943374634, -0.2913571894168854, 0.6226792335510254, -0.8779999613761902, 1.334143877029419, -2.2957921028137207, -0.8311752080917358, 1.0154144763946533, -0.0760437622666359, -0.5261260271072388, -0.4681515693664551, 0.11923468112945557, -0.16482976078987122, -1.2177897691726685, 0.3949640989303589, -0.28941377997398376, -1.2526124715805054, 1.0632474422454834, 0.6164824962615967, 0.3861005902290344, 0.8394358158111572, -1.5021557807922363, -1.3128036260604858, 1.4688831567764282, -0.12628915905952454, -0.7376418709754944, -1.1135470867156982, -1.1427878141403198, -0.534554123878479, -0.8996068239212036, -0.6685958504676819, 0.8302143812179565, -0.7197417616844177, -0.09410719573497772, 0.12025826424360275, 1.0598957538604736, -1.2437704801559448, 0.8277987241744995], [-0.7497372031211853, 0.020768899470567703, -0.4982452690601349, -0.24788786470890045, -0.9706589579582214, 1.1534055471420288, 0.27326497435569763, 0.9919995069503784, -0.22339054942131042, -0.11103427410125732, 2.2846899032592773, -0.8536315560340881, -0.20855095982551575, 0.5026041865348816, 1.5019962787628174, -0.6461896896362305, 0.7410633563995361, 1.2800242900848389, -1.0232696533203125, 0.2974783182144165, 0.5046631693840027, -1.388840913772583, -0.9881691932678223, 1.8843415975570679, -0.5967313647270203, 0.5561569333076477, 0.13025711476802826, 0.07096308469772339, 0.3572627305984497, 0.3637804388999939, 1.772032380104065, -0.863693118095398, 0.11608601361513138, 0.010227564722299576, -0.2884978652000427, -0.76735919713974, -0.6073235869407654, 0.11795850843191147, -0.3663865327835083, -0.2580265700817108, 0.9440988302230835, 0.6587026715278625, 0.6100223064422607, -0.21809761226177216, 0.27190929651260376, -0.6365501284599304, 0.5715307593345642, 2.1492245197296143, -0.41628000140190125, -0.5230109095573425, 0.4138314723968506, 0.4006400406360626, 0.8526360988616943, 0.4240185022354126, -0.1576518714427948, 1.2352646589279175, 0.7832242250442505, -0.1820044219493866, -0.5087327361106873, 1.6149423122406006, 1.3071861267089844, -1.8736660480499268, 1.5767881870269775, -1.6013425588607788, -0.014412076212465763, -1.1536989212036133, -1.028056263923645, -1.165416955947876, 0.4136483073234558, -0.1905938684940338, -0.37425118684768677, 0.04029453918337822, 0.729019284248352, 0.33117571473121643, 0.11535250395536423, 0.9100767374038696, 0.5363033413887024, -0.09066974371671677, 2.3484206199645996, 0.9702959656715393, 1.7363606691360474, -1.3859517574310303, -0.3961060047149658, -0.5435184240341187, 0.16830316185951233, -0.06074826419353485, 1.4575608968734741, 0.7958841919898987, 0.882573664188385, 0.1696040779352188, 1.2177809476852417, -0.49301448464393616, -0.3865666091442108, 0.07170218229293823, -0.3527171015739441, 1.6744229793548584, 0.019288504496216774, -0.48539823293685913, 0.12883147597312927, 0.879690945148468, 0.21031403541564941, 1.7773363590240479, -0.3401544988155365, 0.6154214143753052, -0.2898995578289032, -0.4800534248352051, 0.2711564302444458, -0.2576637268066406, -1.8362957239151, 0.5264521241188049, -0.8928830027580261, 0.8538239598274231, 0.6849604845046997, -2.1879801750183105, 0.16292302310466766, 0.33455461263656616, -0.9036839604377747, 0.529952347278595, -1.6075226068496704, -1.3073878288269043, 0.08549808710813522, -0.1363513320684433, 0.5185369253158569, 0.5870123505592346, -0.21179695427417755, -0.22276125848293304, -0.11768224090337753, -1.8993422985076904], [1.6611613035202026, 0.27454760670661926, -0.8616747856140137, -0.4398934841156006, -0.38228607177734375, 0.7277445197105408, -1.0824323892593384, -1.4573757648468018, -0.3065330386161804, -1.8016127347946167, 0.22524648904800415, -1.4670753479003906, -0.08925991505384445, -0.3392091691493988, 0.6687686443328857, -1.472476601600647, 0.5275208353996277, -0.6114144921302795, -0.13745956122875214, -1.4536939859390259, -0.06740012764930725, 0.7134738564491272, 0.9949864149093628, 0.4389553666114807, -0.6982109546661377, 0.6565921306610107, 0.4685560464859009, 0.5319212675094604, 0.22899116575717926, -0.23016244173049927, 0.9486114978790283, 1.1316944360733032, -0.10067402571439743, -0.29323214292526245, -0.2873443067073822, 0.5056150555610657, -0.21031345427036285, 0.760529637336731, 0.26255083084106445, -0.4943870007991791, 0.7861935496330261, 1.109734058380127, -0.6515089869499207, 0.11239025741815567, -1.2238012552261353, 0.46990612149238586, -1.021134614944458, 0.03451596200466156, 0.9674006104469299, 1.2114886045455933, -0.7032313942909241, 0.33614248037338257, -0.5075384974479675, 0.2887318730354309, -0.3438924551010132, -0.0030019518453627825, 1.7061597108840942, -1.1182345151901245, -0.19862164556980133, 0.6656420826911926, -0.27215564250946045, -2.7495806217193604, -2.014594078063965, 0.07342948019504547, 0.24755500257015228, -0.24425944685935974, 1.967306137084961, -0.7550663352012634, 0.16925403475761414, 0.5453162789344788, 1.2399277687072754, 0.6971776485443115, -1.4918859004974365, 0.11561701446771622, 0.953750729560852, 0.5767499208450317, 1.0338197946548462, 0.009660274721682072, 1.144326090812683, 0.7173191905021667, -0.4401029348373413, 2.289584159851074, 1.0802773237228394, 1.8995261192321777, 1.8456995487213135, -0.11346252262592316, 0.5246151089668274, 0.35665443539619446, 1.7094124555587769, -0.48315978050231934, 1.4142159223556519, -1.2398335933685303, 0.5389122366905212, -0.3747803568840027, -0.028225749731063843, 0.0544021837413311, -0.9354806542396545, -0.8184728026390076, -0.23704098165035248, -0.10343378037214279, -0.4072885513305664, 0.9801688194274902, -0.16954012215137482, -1.1141917705535889, 0.40266847610473633, -2.289980411529541, 1.2771313190460205, 0.2060726135969162, 1.6681785583496094, 0.7994407415390015, 0.28730472922325134, -1.0335084199905396, -0.3476661145687103, -0.45050135254859924, 1.8468453884124756, -0.7803038358688354, -1.0130270719528198, 0.41599249839782715, 0.21002346277236938, 1.0721733570098877, 0.05545240640640259, 2.202354669570923, -0.42565852403640747, 1.8249672651290894, -0.6492951512336731, -0.19871404767036438, 0.9688535928726196, -0.5986543297767639], [1.4066702127456665, -1.5600700378417969, -0.8530480265617371, 0.8934707641601562, 1.3679840564727783, 0.4297671318054199, -0.4279631972312927, 1.2057706117630005, -0.31716468930244446, 0.5558509230613708, -0.9567686319351196, -0.1271539181470871, -1.117632269859314, -0.02881649136543274, 0.19280970096588135, 1.8005684614181519, 0.25341796875, 0.2379802018404007, -1.5868544578552246, 0.037120070308446884, -1.374837875366211, -1.0588535070419312, -0.209535613656044, -0.16604973375797272, 1.9741920232772827, 1.2311865091323853, 0.4205467700958252, 0.0978347510099411, 1.0983034372329712, 0.3328370451927185, -0.188731387257576, -2.117600202560425, 0.18009208142757416, -0.7254273891448975, -0.4755299687385559, -0.5939142107963562, -0.7435917854309082, -0.18805363774299622, 0.08173826336860657, -0.7020349502563477, -0.4208150804042816, -0.1510254144668579, 0.655769407749176, -0.40388572216033936, 0.7305698990821838, 0.8814557790756226, -1.05672287940979, 0.21309611201286316, 0.7571805119514465, 2.086130380630493, 2.0487935543060303, 0.1501472443342209, -0.026096342131495476, 0.08604362607002258, 1.3194372653961182, -1.7140047550201416, -1.2365797758102417, 0.029158344492316246, 1.0907011032104492, 0.17727743089199066, 0.6541743874549866, -1.1013059616088867, 0.5515854954719543, -1.213113784790039, -0.15165671706199646, 1.293415904045105, 1.2702962160110474, 1.1798746585845947, 0.9776555895805359, 1.1978694200515747, -2.0730972290039062, 0.516056478023529, -0.48477551341056824, 0.8143666982650757, 0.3456916809082031, 0.3690970540046692, 2.4101898670196533, 0.20371203124523163, 0.2260202318429947, 1.5201798677444458, -0.4803781807422638, 1.1553518772125244, -0.766377866268158, -0.478497713804245, 0.18973380327224731, -0.7640892863273621, -1.4604978561401367, -0.531696081161499, -1.3840733766555786, 0.03487730771303177, 0.12875241041183472, -1.093432903289795, -1.353597640991211, -0.21879985928535461, -0.2594490945339203, -1.0631322860717773, -0.9809680581092834, -1.0519479513168335, 1.1139171123504639, -0.1628425419330597, 0.2559429705142975, 1.1615186929702759, 2.0807645320892334, -0.6886627674102783, -0.43737858533859253, 0.25965094566345215, -0.9549003839492798, -0.4089813232421875, -0.08874606341123581, 1.301500916481018, 1.420589804649353, 0.23950566351413727, -0.9564938545227051, -0.20397892594337463, 0.9428942203521729, -0.06387756019830704, -2.032460927963257, -0.9672929048538208, 2.0414695739746094, 0.9649145007133484, -0.6509749889373779, 0.5665726065635681, -1.0556410551071167, -0.597812294960022, 0.1416475474834442, 0.2367793768644333, -0.3082880973815918, 1.2397528886795044], [-1.1726137399673462, 0.03616990149021149, 0.2276403158903122, 0.6857153177261353, 0.16639059782028198, 0.7841417193412781, 0.622087299823761, -0.34399479627609253, 0.14748519659042358, -0.9866859316825867, 1.068617582321167, 0.33349478244781494, 0.9837000966072083, -0.3632056415081024, -0.2506186068058014, 0.6153289675712585, -0.4825126528739929, 1.6567256450653076, 0.562950611114502, -0.9125733375549316, -0.43957868218421936, -0.8026522994041443, 0.7484294772148132, -0.6912389993667603, -0.45090949535369873, 1.329390525817871, 0.8263789415359497, -0.6463264226913452, 2.9631521701812744, 0.7040735483169556, 0.5820746421813965, 0.7787388563156128, 0.20264656841754913, -0.41079023480415344, -1.3089728355407715, -0.8892883658409119, -0.5080087184906006, -0.7354444265365601, -0.16190044581890106, -0.16211684048175812, 2.3753750324249268, 1.9977747201919556, -1.5796276330947876, 0.3178984820842743, -2.065349817276001, 0.6920856237411499, -0.6202725172042847, -0.5122167468070984, -1.956885814666748, -0.5753592848777771, 1.3696480989456177, 1.1793361902236938, 0.8247647881507874, 0.6537902355194092, -0.29566702246665955, -1.122719168663025, -0.11258514970541, 2.0664432048797607, 0.06016841158270836, 0.48124733567237854, -0.5242534279823303, -0.43923506140708923, -0.3505937159061432, -0.8904129862785339, 1.1753655672073364, 1.6634615659713745, -0.1994473934173584, 0.7187180519104004, 1.4334633350372314, -0.3490099310874939, -0.189827561378479, 0.023974668234586716, -0.4568924903869629, -1.2661011219024658, 0.16300275921821594, -0.18535470962524414, 0.0557464063167572, -0.30594006180763245, 0.48642417788505554, -0.7790377140045166, 0.642889678478241, -0.3769674599170685, -1.732813835144043, 1.4001288414001465, -0.27758345007896423, -1.008286952972412, -0.6426932215690613, 0.09921237081289291, -2.2806296348571777, 1.018033504486084, 0.78961181640625, -1.1760140657424927, -0.18003927171230316, 1.2340261936187744, -0.16901400685310364, -2.0340957641601562, -1.1095589399337769, -0.08291592448949814, -0.22457031905651093, 0.2790677547454834, 0.4120752215385437, 1.9377586841583252, 0.350851446390152, 0.7145500183105469, -0.8668699860572815, -1.2412021160125732, 0.20471040904521942, -0.9075828194618225, 0.830832302570343, 0.39296582341194153, 1.540667176246643, -0.34465253353118896, 1.6747864484786987, 1.3902955055236816, 1.2881875038146973, -0.14884313941001892, -2.683979034423828, 0.7977630496025085, -0.1697336882352829, -0.6865692138671875, 2.395920515060425, 0.6412568092346191, -0.8367229700088501, 0.04410610720515251, -1.633581519126892, 0.026146724820137024, -0.012538906186819077, -1.0551666021347046], [1.705573558807373, 0.3328877389431, 0.6837785243988037, -1.3270920515060425, -1.4720107316970825, -0.5139620900154114, 0.6533512473106384, -0.24016012251377106, -1.1981487274169922, -1.4966667890548706, 0.7322507500648499, 0.1343095898628235, 0.2530372738838196, 1.1439818143844604, 2.9419238567352295, 0.44387826323509216, -0.22997267544269562, -0.6905052661895752, 0.054335977882146835, 1.3155497312545776, -0.9697125554084778, -0.4870404899120331, -0.6459605097770691, -0.3824754059314728, 0.6432668566703796, -1.2911503314971924, 0.11950182169675827, 0.2718378007411957, -0.4907180070877075, -0.32596898078918457, 0.39518237113952637, 1.0297194719314575, 1.4498865604400635, 0.0419122688472271, 0.46977517008781433, -0.359698623418808, 0.13180918991565704, 0.009092897176742554, -0.024432417005300522, 0.4323274791240692, -0.7226628065109253, 1.2016396522521973, 0.41281941533088684, 1.1670314073562622, -1.1507642269134521, -1.431128740310669, 0.43808457255363464, -0.4435292184352875, 0.2629442512989044, 0.9650930166244507, -0.20774434506893158, -0.24810083210468292, 0.7756487131118774, 0.4467051029205322, -1.3400118350982666, -0.4908388555049896, -0.49523118138313293, 0.9601767659187317, 0.6103543043136597, 0.11189482361078262, -1.5045444965362549, -0.8335727453231812, 0.1062355637550354, -1.0002566576004028, 1.114168643951416, -2.196223020553589, -1.4233802556991577, 1.9087682962417603, -0.11139164865016937, 0.319439560174942, -0.5234360694885254, -1.001319408416748, -0.886688768863678, -0.5792179703712463, -1.1126147508621216, 0.4128602147102356, 0.5076548457145691, 0.07802903652191162, 1.3817119598388672, -0.37988296151161194, -0.4318879246711731, -0.0212019644677639, 2.132474422454834, 0.8998541235923767, -1.5013842582702637, 2.2608323097229004, -0.14303059875965118, 2.1666629314422607, 0.38752931356430054, 0.9641902446746826, 0.9037487506866455, 0.6023859977722168, -0.17174139618873596, -0.8225910067558289, -1.113370656967163, 0.16875840723514557, 1.365899682044983, -1.421290636062622, -0.6054785251617432, 0.3100733757019043, 0.7486861944198608, -1.5233395099639893, -0.16638989746570587, -0.657257616519928, -1.2297075986862183, 1.4721640348434448, 0.8883671164512634, 1.380977988243103, -0.6762915849685669, -0.5420098900794983, -1.4225833415985107, -0.13540390133857727, 0.21292290091514587, 0.46044808626174927, -0.7656480073928833, -0.22064025700092316, -0.8764063119888306, -2.3469443321228027, -0.7130388617515564, 0.6587048172950745, 0.5714683532714844, -0.5886181592941284, -0.10457241535186768, 0.12411856651306152, -0.35856661200523376, 0.4586373269557953, 0.8935689330101013, -1.3908874988555908], [0.2779899537563324, 0.12399481236934662, -0.08072568476200104, -0.9621537923812866, 1.5581845045089722, -0.6996275782585144, 0.01617586798965931, -0.1423673778772354, -1.473017930984497, 0.5026914477348328, 0.39040666818618774, -0.16264526546001434, 0.19340571761131287, -1.4797683954238892, -0.9118058681488037, 2.205519676208496, -1.0120567083358765, -0.9207714200019836, 0.04114184528589249, 1.00640869140625, -0.3509208559989929, 0.16739702224731445, -0.35610833764076233, -0.4303056597709656, -1.11651611328125, 1.404002070426941, 2.349329948425293, 0.12943720817565918, 0.16903255879878998, 0.18878647685050964, -1.055906891822815, 1.8719133138656616, -0.8905395269393921, 1.1685868501663208, -0.3699711263179779, 1.4922304153442383, -0.5249174237251282, -0.4854266941547394, -1.1520729064941406, -1.1039583683013916, -0.14015501737594604, -0.5844482183456421, -1.0818589925765991, 0.8840346932411194, -0.686227560043335, -0.04805842787027359, 1.125749945640564, 0.5594143867492676, 1.1954509019851685, 0.265363484621048, 0.5232107043266296, -1.5790369510650635, 0.14203551411628723, -0.16837064921855927, -1.3121237754821777, 0.05805171653628349, -1.9352936744689941, 2.9202089309692383, -0.19870001077651978, -0.08926226943731308, 0.2620907127857208, 0.09099184721708298, -0.2357722967863083, 1.1564135551452637, 0.36643725633621216, 0.6313817501068115, 1.6258518695831299, 0.3177926242351532, 1.1688201427459717, 0.6333438158035278, -0.18428879976272583, -0.870491623878479, -0.6963435411453247, -0.39660245180130005, -0.10208889096975327, -0.8897721767425537, -0.09283177554607391, -0.7652561664581299, -1.5224237442016602, 0.034945983439683914, -2.015547752380371, 0.6723306179046631, -0.1507142335176468, 2.068366289138794, 1.2694872617721558, -0.38283947110176086, -0.3587891161441803, -0.4507673680782318, -0.8230783343315125, -0.1558685302734375, -0.48288899660110474, 0.009677969850599766, 1.3872755765914917, 0.7327951788902283, 0.8835290670394897, -1.1274877786636353, 1.0989024639129639, -1.7378361225128174, -1.9923537969589233, -0.43265604972839355, 0.08360373973846436, -0.34588584303855896, 0.6295680999755859, -0.63059401512146, -0.4793980121612549, -0.14960330724716187, 0.547197163105011, 0.3597118854522705, -1.0130482912063599, -1.5111571550369263, 2.4167773723602295, -0.6849002838134766, -0.266470342874527, -0.30609384179115295, -0.5011794567108154, 1.0056687593460083, -0.5556548237800598, -1.7988569736480713, 0.11716028302907944, -0.06756765395402908, 0.9327123165130615, 0.1229969710111618, -0.297782838344574, -0.2147495448589325, -0.6482835412025452, -0.4737589359283447, -1.429561734199524, 1.482162594795227], [0.7688900232315063, -1.519250750541687, -0.14927300810813904, 0.4475362300872803, -0.9282896518707275, 0.27693814039230347, 0.20985250174999237, -0.4364044666290283, -0.35004734992980957, -0.10113043338060379, 0.46031010150909424, -0.983162522315979, 0.8051475882530212, 1.7757482528686523, 1.3127907514572144, 0.045170508325099945, 1.8950310945510864, 1.514037013053894, -0.644731342792511, 1.8795478343963623, -0.752263069152832, 1.9628421068191528, 0.7334738969802856, -0.1833200603723526, 0.2981320023536682, -0.8428305387496948, -1.9864684343338013, 0.500972330570221, -1.3899134397506714, -1.1793272495269775, 0.09582889825105667, 0.6427253484725952, -0.7575328946113586, 0.014166787266731262, -1.7269326448440552, -0.22387109696865082, 1.3115853071212769, -2.0075409412384033, -0.1579234004020691, -1.0950864553451538, 0.6308737993240356, 0.20425467193126678, 0.11958315968513489, 0.9012932181358337, -0.4475289583206177, 0.014572816900908947, 0.11299755424261093, -0.5165515542030334, 1.4481817483901978, 0.9613769054412842, -1.0083733797073364, 0.5897483229637146, 0.3383072018623352, 1.8488824367523193, -1.228847622871399, -0.394247442483902, 0.36688870191574097, 2.027893304824829, -1.5645081996917725, -0.19218836724758148, -0.7670560479164124, 0.3635261356830597, 0.3435949981212616, 2.042731285095215, 0.4104204475879669, 1.1672345399856567, -1.2810568809509277, 0.2113049328327179, 1.361283779144287, 0.39935559034347534, 0.037272900342941284, 0.8714231848716736, 1.7290136814117432, 0.791027307510376, -1.2733497619628906, -1.0169565677642822, 0.668252170085907, -0.7442456483840942, 0.0009441530564799905, 0.8617039918899536, 1.0299052000045776, 2.0367848873138428, 0.5922536849975586, 1.934175729751587, 0.7645523548126221, 0.5554310083389282, -0.33979156613349915, -0.3610549569129944, -0.8706512451171875, 0.2660364806652069, -0.5196660161018372, -0.7373148202896118, -1.21595299243927, 0.2750106751918793, 0.818324089050293, -1.1049916744232178, -0.7436965703964233, 1.7966351509094238, 0.8593897223472595, 2.5372369289398193, -0.9884138107299805, 0.7447780966758728, -0.5156227946281433, -1.0195186138153076, -0.543632984161377, 0.9690225124359131, 0.5310842394828796, 1.0597420930862427, -1.3354301452636719, -0.6757020354270935, 1.375152826309204, 0.6445749402046204, 0.07626701891422272, -2.4851231575012207, -0.36947768926620483, 0.5004541873931885, 0.7772210836410522, 1.959035873413086, -0.5655434131622314, -0.5970041155815125, -2.017228603363037, -0.8832063674926758, 0.7296742796897888, 1.0724105834960938, -0.5953763723373413, -1.301338791847229, 0.4766784906387329, -0.7562096118927002], [-0.8652184009552002, -1.3025078773498535, 2.0276577472686768, -0.44164299964904785, -1.2258915901184082, 0.10678229480981827, 0.18688322603702545, -1.0277740955352783, 0.44396933913230896, -1.1337342262268066, 0.8710108995437622, 0.20629052817821503, 0.21105772256851196, -0.7525407075881958, 1.0627154111862183, -0.5215527415275574, -0.23219077289104462, 1.2712221145629883, -0.4045295715332031, -0.5787876844406128, 1.092974066734314, -0.5892412066459656, 0.3548045754432678, -0.054969679564237595, 0.3339749872684479, 0.5903589129447937, 0.6618146300315857, 0.06749264895915985, -0.23872104287147522, -0.6896218061447144, -1.0382364988327026, 0.07902543246746063, -2.348618984222412, 0.7053989171981812, -0.17981113493442535, -1.2404483556747437, -0.2915663421154022, -1.89643394947052, 0.12997578084468842, -1.1674765348434448, 0.25170835852622986, -0.6718156337738037, 0.1764189898967743, -1.1020101308822632, 1.1985052824020386, 1.595828890800476, 0.12355343252420425, -0.39333081245422363, -0.3618377149105072, -0.6628758311271667, -0.5669334530830383, -0.8082807064056396, 0.13710527122020721, -0.29669904708862305, -0.5469157099723816, 0.6985392570495605, -0.6158304810523987, 1.0252549648284912, 0.95308518409729, 0.751991868019104, -2.1967833042144775, -0.5370402932167053, 0.8387394547462463, -1.509095549583435, -0.3027481734752655, 1.210660457611084, 2.4025039672851562, 0.8925265073776245, 0.01856282167136669, -1.7245842218399048, -1.3774093389511108, -1.3513274192810059, -1.0591769218444824, 0.8906527161598206, -1.5070395469665527, 0.2844575345516205, 0.11927014589309692, -0.001925310236401856, 1.9854605197906494, 0.9903701543807983, -0.8374748826026917, 0.08897370845079422, 2.4983935356140137, -0.3789542317390442, 0.428026407957077, -0.9900375604629517, -1.6064740419387817, 0.04095190390944481, -2.371980667114258, 1.865620493888855, 1.3934619426727295, 0.010370595380663872, 0.6323062181472778, 0.670297384262085, 1.943851351737976, 0.2838115990161896, -0.2543413043022156, 2.2085461616516113, -0.6131580471992493, 0.6442369222640991, 0.34607449173927307, -0.5086621046066284, -0.044923290610313416, 0.07665816694498062, -0.517182469367981, -0.36671552062034607, -1.0306878089904785, 0.3043346405029297, -2.2368967533111572, 0.1124735027551651, 0.6718354821205139, 0.26694440841674805, -0.2645612061023712, -1.7687283754348755, 0.6091997623443604, 1.0972036123275757, 0.445222407579422, -0.38407284021377563, 1.6404238939285278, 1.0650273561477661, -0.41645240783691406, 0.4658675491809845, 0.13016371428966522, 0.8142763376235962, -0.8783025741577148, -0.33588019013404846, 1.668960452079773, -0.0010911066783592105], [0.2829523980617523, 1.409751296043396, -0.38385009765625, -2.155599594116211, -0.657363772392273, 0.07734070718288422, 0.8340303301811218, -0.7098863124847412, 0.16601026058197021, 0.725423276424408, -0.5674207806587219, -0.36014324426651, -0.6794856190681458, -1.4804174900054932, -0.10802499949932098, -0.06256361305713654, -0.4332694113254547, -0.4406983554363251, 0.15535247325897217, -0.00016978630446828902, -0.77203369140625, -1.002264380455017, -0.1560918688774109, 1.3023401498794556, -0.5401103496551514, 0.1362321674823761, 0.6659180521965027, -2.3459300994873047, 1.0976300239562988, 1.0424630641937256, 0.759845495223999, -0.5671975612640381, 0.84760582447052, 0.20165874063968658, -0.07229804247617722, 0.30292099714279175, 0.8030309081077576, 0.3933986723423004, -2.0410549640655518, 0.5674381852149963, -3.1088736057281494, 1.3713942766189575, 1.532128095626831, -0.47038429975509644, 1.5340371131896973, -0.12134502083063126, -0.34995272755622864, 0.09397423267364502, -1.365237832069397, -0.47969648241996765, -0.4033275842666626, -0.4633907079696655, -1.2703964710235596, -0.8857508301734924, -1.3523722887039185, 1.0447783470153809, -0.5738852024078369, 2.007413864135742, -0.15738177299499512, 2.1532533168792725, -0.6846428513526917, -1.192970633506775, 0.6118695735931396, 0.10985884815454483, 0.5055024027824402, -0.6274812817573547, 0.7449011206626892, 2.282595157623291, 0.5497097969055176, 0.12851229310035706, 2.567938804626465, 0.44870448112487793, -0.04481368511915207, -0.40244951844215393, -0.5115847587585449, 0.32257363200187683, 1.2359082698822021, 0.8951559662818909, -1.9258264303207397, 0.39528167247772217, -1.2080706357955933, -0.8160752654075623, 2.1968023777008057, -0.9090666770935059, -0.3450075089931488, 0.7920196056365967, -0.5677002668380737, 0.43555906414985657, -0.6687411069869995, -0.843072235584259, 1.1127370595932007, 0.6663880348205566, 2.271141290664673, 0.4293338358402252, 0.0699828490614891, -0.6952588558197021, -1.737029790878296, -1.6415833234786987, -0.4840250611305237, 1.0473510026931763, 0.8081072568893433, 0.2551625669002533, 1.3883978128433228, 0.21317198872566223, 0.4954814314842224, 1.2169976234436035, -1.0897281169891357, 0.8526005744934082, -0.13764970004558563, -1.2921894788742065, 0.0981246680021286, 0.408950537443161, -1.269175410270691, 0.15413357317447662, 1.3430010080337524, 0.5213218927383423, 0.7357739210128784, -2.029762029647827, 0.5836868286132812, -0.3266289532184601, 1.6779847145080566, 0.9968904256820679, -1.3039730787277222, -0.06360746920108795, 0.8865091800689697, -0.18481943011283875, 0.6492922306060791, -0.6655501127243042], [-1.109548807144165, 0.5037245154380798, -0.21138131618499756, 1.6554840803146362, -0.9855605959892273, -0.19332006573677063, 0.503609299659729, -1.017783761024475, 0.007105634082108736, 0.7974500060081482, 0.2865208089351654, 0.14467008411884308, -1.2116222381591797, -0.14687958359718323, -1.113371729850769, -1.2627003192901611, 0.4154277741909027, 0.42257654666900635, 1.656187891960144, -1.6969115734100342, 0.7420295476913452, -1.8340790271759033, 0.4519021213054657, 0.8163963556289673, 0.41389626264572144, -0.28464221954345703, -1.0614585876464844, -0.6290063858032227, -0.09515312314033508, -0.9620879888534546, -1.323952555656433, -1.4863916635513306, 0.6221200227737427, -0.0955924317240715, -0.13163252174854279, -0.6197255253791809, -0.22775304317474365, 0.4402996599674225, 0.2238868623971939, 0.32352226972579956, 0.40060529112815857, -0.2817515432834625, 1.486581563949585, 0.38100308179855347, -0.5989633202552795, 1.7352463006973267, 0.1684243232011795, 1.318008303642273, 0.012675235979259014, 1.1623210906982422, 2.2995896339416504, 0.5152392983436584, -0.2385733276605606, -0.6186042428016663, -0.4197527766227722, -0.798997163772583, 0.3042850196361542, 0.48465654253959656, -0.9920664429664612, 1.2784730195999146, -0.2559778094291687, 0.5533674955368042, 1.2056305408477783, -1.210440993309021, -0.7213240265846252, -1.2066646814346313, 2.2037601470947266, 1.506463646888733, 2.357297658920288, 0.607501745223999, 0.14612098038196564, 0.09685961157083511, 0.8091748356819153, -0.6007703542709351, -1.8384718894958496, -1.5039889812469482, -1.870680332183838, -0.011290977708995342, -0.47630050778388977, 0.34935033321380615, -0.7687957882881165, 1.9764598608016968, -0.16955047845840454, 0.9922041893005371, 1.7860300540924072, -1.1128112077713013, 0.6525231599807739, -1.6573407649993896, -0.9504740834236145, 0.4115927815437317, 0.03397604823112488, 0.27303412556648254, -0.930554986000061, 2.2132887840270996, 0.7748133540153503, -0.32416772842407227, 1.5946483612060547, -1.3871060609817505, 0.5001316666603088, -0.6751644611358643, -0.16716161370277405, -2.1787948608398438, 0.27461346983909607, 0.08319318294525146, 0.5177889466285706, -1.7110283374786377, 0.7989671230316162, -0.21426022052764893, -0.8558897972106934, -0.07507357746362686, -0.4660665690898895, 0.11223573237657547, 0.5399727821350098, 1.7202317714691162, 0.24454215168952942, -0.663680911064148, 0.07973264902830124, -0.881132185459137, -1.7431392669677734, 1.268006443977356, -0.3822450637817383, 0.9085100293159485, -0.5321052670478821, 0.36578884720802307, 1.0763282775878906, -0.5084636807441711, 1.0895426273345947, 0.4793144166469574], [-0.13824012875556946, -0.6236432790756226, 1.8731263875961304, -2.025477409362793, 0.355569064617157, 0.2035660743713379, -0.6309971809387207, -0.36178064346313477, -0.31040266156196594, -1.1994880437850952, -0.7363905310630798, 0.35754120349884033, -0.6984764337539673, -0.5748593807220459, 1.8508220911026, -0.34990808367729187, 0.5595781207084656, -0.5936506390571594, 1.0741791725158691, 0.6672895550727844, -0.6362127661705017, 0.36675605177879333, -0.6322656869888306, 1.7610483169555664, -0.3247397243976593, 1.0139826536178589, -0.7812045812606812, -0.33935293555259705, 1.0438542366027832, 0.17771050333976746, -0.39027491211891174, 0.4785287380218506, -0.804312527179718, 0.26983317732810974, 1.8345102071762085, -1.6961268186569214, 1.112380027770996, 0.034965042024850845, 0.6585492491722107, 0.7700450420379639, 0.18016411364078522, 0.3266383111476898, -0.05000729858875275, -0.6177929639816284, -1.7807917594909668, -0.6573780179023743, -0.10911595821380615, -0.5245906114578247, 0.9912651777267456, 0.14977401494979858, 1.4266059398651123, -1.377277135848999, -2.484408140182495, -0.9422338604927063, -0.39868199825286865, -1.6038225889205933, -0.15676291286945343, -0.5801101922988892, 0.4770190119743347, -0.3146389126777649, -0.9842071533203125, -0.29566606879234314, -0.42142364382743835, 0.479708194732666, 0.19980503618717194, -0.8588428497314453, 1.3093452453613281, 1.1256438493728638, -0.08680176734924316, 0.5076740980148315, 1.596805453300476, 0.3587501645088196, 0.5652275085449219, 0.49876877665519714, -0.42106157541275024, -1.5734223127365112, 0.13860934972763062, -0.969156801700592, 0.2741253972053528, -1.957756757736206, 0.3561290502548218, 0.6779357194900513, 0.6015066504478455, -0.2873084247112274, 1.266445279121399, -0.9361401200294495, 0.5479217767715454, 0.1744958460330963, -1.1635428667068481, 0.9048059582710266, 0.7078237533569336, 0.06072906404733658, -1.3501403331756592, -0.18382839858531952, 0.07436036318540573, -0.6688230037689209, -0.5388957858085632, 0.01681826077401638, -0.4772931635379791, -0.08423267304897308, 1.0118590593338013, 0.21525970101356506, 0.40641242265701294, 0.659523606300354, -0.68939608335495, 0.5290457010269165, 0.40554022789001465, -0.3530423641204834, 1.3578834533691406, 0.2227255403995514, 1.879406213760376, -1.5385719537734985, -0.9347366690635681, -1.2366430759429932, 2.3327295780181885, -1.8353638648986816, 0.03212623670697212, 0.6988247632980347, -0.45648109912872314, -0.6359574198722839, -1.7414757013320923, 0.1483328491449356, -1.9264439344406128, -0.5616359114646912, 0.08385786414146423, 0.12596803903579712, 1.4694262742996216, -1.8955016136169434], [0.006898972671478987, -0.9366902112960815, 0.20628802478313446, -0.18099714815616608, 0.11755242198705673, 2.4528064727783203, -0.13990607857704163, 0.9067209959030151, -1.2096292972564697, -0.2915090024471283, 0.541398286819458, 1.8455814123153687, -0.004531294107437134, 1.3127496242523193, -0.372591495513916, -0.23111163079738617, 1.4330615997314453, 0.17836841940879822, -0.2677774429321289, 1.0299402475357056, 0.5523893237113953, 0.6657067537307739, 0.8561261892318726, 1.390733242034912, -0.1823151856660843, -0.6674240827560425, 0.7243112325668335, -0.34231650829315186, 1.1245841979980469, -0.8360812067985535, 1.1644290685653687, -0.6716720461845398, -0.6877070665359497, 0.6032411456108093, 0.13705477118492126, -0.5896507501602173, -0.11057668179273605, 0.42650070786476135, 1.393757700920105, 1.62144935131073, -0.2365415394306183, 1.3984626531600952, 0.13903096318244934, -0.46684983372688293, -0.17094630002975464, 0.014656594023108482, 0.6097854375839233, -0.7078230381011963, 2.0881214141845703, -0.1174592599272728, 0.0027999808080494404, 0.4303348958492279, -0.07672677934169769, 1.8603382110595703, 0.7948001027107239, 1.0048922300338745, -1.5912433862686157, 0.3272806406021118, 2.5374834537506104, 0.15674304962158203, 1.0554031133651733, -0.8441285490989685, 1.3743410110473633, 0.5000596046447754, 0.5731101036071777, -0.2529686689376831, -0.3785392642021179, 0.9330712556838989, 0.8530306816101074, -0.4317358732223511, -0.20014119148254395, 0.05461704358458519, -0.5837363600730896, 0.3112331032752991, 2.295074462890625, -1.4320553541183472, 1.8806825876235962, 0.7070780992507935, -0.5944324135780334, 1.147108793258667, -0.3666037321090698, -0.7797318696975708, -0.09738321602344513, 1.2456278800964355, 0.14316023886203766, 1.256921410560608, -2.115015745162964, -0.10211780667304993, -0.631485641002655, -1.7752842903137207, -0.06037002429366112, -0.07372613996267319, -0.9572827816009521, -0.6491747498512268, -1.1417702436447144, 1.2810883522033691, 0.8145369291305542, -0.6000768542289734, -0.32409340143203735, -0.7252498269081116, 0.8128786087036133, 0.008547293022274971, -1.8868838548660278, 0.09637649357318878, 0.2536928653717041, 0.20755957067012787, -0.7184447646141052, 1.8983722925186157, 0.5894224643707275, 0.2190379649400711, 0.5097314715385437, 1.600434422492981, 0.32410919666290283, 0.15252822637557983, -0.053728848695755005, -0.6795122623443604, 0.7614744305610657, -0.7239494919776917, -0.5518864393234253, -0.01580028235912323, 0.1180163323879242, -0.06848996877670288, -1.4621796607971191, -0.6572047472000122, -1.6968427896499634, 2.5183768272399902, 1.5263235569000244, 0.10840823501348495], [1.245985746383667, 0.8224339485168457, 0.05023574084043503, 1.034501075744629, -0.13084954023361206, 0.8432925939559937, -0.6328747868537903, 0.06506690382957458, 0.45762205123901367, 2.7074055671691895, 0.06449328362941742, -0.6433879137039185, 1.4464987516403198, -0.6166985034942627, -0.7489277124404907, -0.16184742748737335, -1.5662420988082886, 0.405396044254303, 0.19539572298526764, -0.08725375682115555, -0.385847270488739, -1.1704813241958618, 0.6311340928077698, -0.8475723266601562, -1.5622572898864746, -0.05264921486377716, 0.07353197038173676, -0.34778934717178345, 0.4066813886165619, -0.7802730202674866, -2.093743324279785, -0.311440110206604, -0.7302749752998352, -0.038812268525362015, 0.13144105672836304, 0.3126733601093292, -1.1831791400909424, 1.0610350370407104, -0.5235282182693481, -2.194789409637451, 0.5274530053138733, 0.271406352519989, 1.0112005472183228, 1.9146918058395386, -1.245855450630188, 1.102526068687439, 0.211227148771286, -0.6017172336578369, 1.3232698440551758, -0.24037450551986694, 0.6040043830871582, 0.38325732946395874, -1.1322882175445557, -0.5360875725746155, 1.0922640562057495, -0.1588248461484909, 1.0802483558654785, 0.27522778511047363, -0.7973790168762207, 0.5776055455207825, 0.33989736437797546, 0.8062697649002075, 0.5229179263114929, -0.2896060049533844, -0.7876690030097961, 0.5438280701637268, -1.171398401260376, 0.2469751536846161, -0.030250560492277145, 0.4099124073982239, -0.21170684695243835, 0.3311634957790375, -1.2505276203155518, -0.2180621474981308, 0.8683530688285828, -0.4856693744659424, 0.1914369910955429, -0.6008977293968201, 0.3369409441947937, 0.5295847654342651, 0.09381098300218582, -1.812427043914795, -1.2312076091766357, 0.6775239706039429, 1.8533780574798584, -1.509889841079712, 0.07466947287321091, 0.4580405354499817, 0.8644402623176575, 0.21435044705867767, -0.6015608906745911, -0.5624609589576721, -1.56314218044281, -0.07139945030212402, -0.3010666072368622, 1.6056979894638062, 1.0997339487075806, 1.5921261310577393, -1.1833746433258057, 0.8920708298683167, -0.5290278792381287, -0.6177178621292114, -0.8912904262542725, -1.188346266746521, -1.1103944778442383, 1.3405003547668457, 0.6979144215583801, -2.0195670127868652, 0.09267640113830566, -0.14108099043369293, 0.3068147301673889, 0.7284885048866272, -0.9474674463272095, 0.02387341484427452, -0.2692553699016571, 1.0007479190826416, 1.264705777168274, 2.519178628921509, 0.6059780716896057, 0.5470141768455505, 0.19388189911842346, -0.24700121581554413, 0.1463858187198639, 0.0021647990215569735, -0.4112452566623688, 1.5207958221435547, -0.04418526962399483, 0.1814921647310257], [0.06039983406662941, 0.5148623585700989, 1.0456578731536865, 1.099137783050537, 0.7433632612228394, 0.24162831902503967, 0.8857450485229492, 1.868542194366455, -0.03851870447397232, -0.3277965188026428, -0.7063629627227783, -1.7967133522033691, 1.0848479270935059, 2.2356653213500977, -0.6547008752822876, 0.019033921882510185, 1.7452664375305176, 1.850892186164856, -0.5184300541877747, 0.7483748197555542, 1.418215036392212, -1.3037893772125244, -0.12275150418281555, -0.12072785943746567, 0.02007768675684929, -0.46967488527297974, 0.18859261274337769, -1.1826571226119995, 0.26646649837493896, -1.3857370615005493, -1.4432806968688965, 0.567203164100647, 0.020060909911990166, 0.911746084690094, -0.13164447247982025, -1.6276947259902954, 0.40445488691329956, 0.5387449860572815, -1.5447883605957031, -0.15032435953617096, -0.28921395540237427, 0.21640101075172424, 1.7883445024490356, 0.9819422960281372, 0.7836242318153381, 0.17447875440120697, -0.8577305674552917, -0.04973727464675903, 0.8522387146949768, -1.3389168977737427, 0.1250825822353363, 1.5267791748046875, -1.062552809715271, -1.7063333988189697, 0.4202330410480499, -1.0583380460739136, 2.6116440296173096, -0.6345529556274414, -1.0997051000595093, -0.3025435209274292, 0.18421560525894165, -1.0314501523971558, 1.2979514598846436, 0.9590157270431519, -0.9728499054908752, -1.1474848985671997, 0.5906428694725037, 0.13540448248386383, -0.5510192513465881, -1.0546605587005615, -0.5312573313713074, -0.039294783025979996, 1.022036075592041, 0.04745658487081528, 0.02622557058930397, 0.35364001989364624, -1.17306387424469, 0.1327899992465973, -0.1317264288663864, 0.6003299355506897, 0.8896791934967041, -0.47333189845085144, 1.1429494619369507, 0.2786897122859955, -0.3442441523075104, 0.6840417385101318, -1.0288374423980713, -1.264035940170288, 0.8500348329544067, 1.277319073677063, 0.6613701581954956, 0.22564542293548584, -1.660696268081665, -0.8202853798866272, 0.5508791208267212, 0.8607025146484375, 1.7400144338607788, 0.40343770384788513, 1.5060678720474243, -2.562649965286255, -0.6559869647026062, 0.09278348833322525, 2.393617868423462, -0.9300442934036255, 0.6751897931098938, -0.7417855858802795, -0.24584992229938507, 0.49309223890304565, -1.2133581638336182, 0.3914440870285034, 0.8134040236473083, 1.2065351009368896, -1.3668413162231445, 1.8092799186706543, 0.31529149413108826, -0.18301557004451752, -0.1270982176065445, -1.5778729915618896, -1.8347840309143066, 1.733243703842163, -0.10927610099315643, 0.18889394402503967, -0.7218779921531677, 0.5355440974235535, 0.27428048849105835, -0.10125922411680222, 2.9176156520843506, 0.019780900329351425], [-0.36197227239608765, 0.2759121358394623, 0.41499876976013184, -0.154571071267128, -0.33909091353416443, 0.8731769919395447, -2.634822368621826, -0.21472755074501038, 1.7628237009048462, -1.279331088066101, 0.06510278582572937, 0.9006224870681763, -0.23839832842350006, -0.056590355932712555, 0.6440843343734741, -0.5800577998161316, 2.231891632080078, -0.19217872619628906, -1.1038856506347656, 1.456201195716858, 0.08215991407632828, -0.19070833921432495, -1.4613205194473267, 0.04697893559932709, -0.5629817247390747, 2.777038812637329, -1.147079586982727, -0.7275455594062805, 1.165299415588379, 0.4550614655017853, -0.5599344372749329, -0.5889617204666138, 0.4726971685886383, -0.10705451667308807, 0.2724737823009491, 1.147363543510437, 1.564773678779602, 0.03911396488547325, 2.1471822261810303, 1.1172034740447998, 0.9522812366485596, -0.29050201177597046, 0.052594710141420364, 0.010129814967513084, 1.0728024244308472, -0.3285391926765442, -1.1279585361480713, -0.011100678704679012, -1.1654969453811646, -0.679211437702179, 1.7682812213897705, 1.6758594512939453, 1.5168861150741577, 0.44880378246307373, 0.4884052574634552, 2.104816198348999, -0.11107838898897171, -0.7228589057922363, 0.5466212630271912, 0.9627138376235962, 0.3987405300140381, 0.7530248165130615, -0.066083624958992, -0.04705885052680969, 1.5113990306854248, -0.9992655515670776, 0.01881979964673519, 0.6638233065605164, 1.7171893119812012, 1.1661298274993896, 0.0849640741944313, 0.4647621512413025, 0.3870787024497986, -1.1251555681228638, -0.5234542489051819, -1.3299020528793335, 1.1513651609420776, 1.6093932390213013, -1.8787341117858887, -0.6420812606811523, -0.32379984855651855, 0.44057774543762207, 1.1365245580673218, -0.15803393721580505, -0.014292122796177864, 0.15189355611801147, -0.5544903874397278, -0.5762672424316406, -2.0339508056640625, -0.42036527395248413, -1.254637360572815, -0.8038718104362488, -0.17653660476207733, 1.6306942701339722, 0.2520360052585602, -1.4017672538757324, -0.9876916408538818, -0.4415554106235504, -0.24509818851947784, 0.08164471387863159, -0.6200768351554871, -1.300667643547058, -0.7318315505981445, 0.13236366212368011, 1.3340210914611816, 0.14709687232971191, 1.6227043867111206, 0.7539213299751282, -0.12825830280780792, 0.4745849668979645, 0.9102305173873901, -1.6236577033996582, 1.1353610754013062, 0.5934917330741882, -0.017902350053191185, 0.36290106177330017, 0.7628192901611328, -0.7017068862915039, 1.8113847970962524, -1.1395909786224365, 0.0009004785097204149, -0.798536479473114, -1.2407511472702026, -0.36424434185028076, -0.665532112121582, -0.4169163405895233, 0.10257195681333542, -0.5208840370178223], [0.37481239438056946, 0.7393807768821716, -0.05060875043272972, -1.0409053564071655, -0.02635345421731472, 1.0427225828170776, 0.48824959993362427, 0.5366032719612122, -0.08529666066169739, -0.8509952425956726, -0.42698121070861816, -0.8468114733695984, 1.3812191486358643, -0.47021302580833435, 0.46192023158073425, -0.8212835192680359, -1.73673677444458, -1.9333393573760986, -0.5932973027229309, -0.9568783044815063, -0.5269362330436707, -0.9243955016136169, -0.24651378393173218, -1.1261836290359497, -1.253774881362915, -0.8706455230712891, 0.16892388463020325, -0.4157739579677582, -0.2372763305902481, -0.3138226568698883, 2.6513540744781494, -1.8019616603851318, 0.8157463669776917, -0.6599823832511902, -1.17979097366333, -0.9896227121353149, -1.9554367065429688, 1.3398247957229614, 1.89993417263031, -1.288777232170105, 0.13961409032344818, -0.9564326405525208, 1.0583367347717285, -0.4579560160636902, 1.6659348011016846, -0.8351659774780273, -0.5019871592521667, 1.5497994422912598, -0.4866016209125519, 0.5928120613098145, 0.1851060688495636, 0.660219132900238, 0.7527068853378296, 1.0626918077468872, -0.2784678041934967, -0.27391496300697327, 0.7548210024833679, 0.6359609961509705, 1.3936121463775635, 0.13262934982776642, 0.17187772691249847, 1.805724024772644, 0.4007536768913269, 0.6064054369926453, 0.7365120649337769, -0.6184512972831726, 0.3370940089225769, 0.6245577335357666, 0.488386869430542, -1.360787034034729, -0.23865002393722534, -0.21685858070850372, -0.8407602906227112, 0.8654202222824097, -0.7519369125366211, -1.096961259841919, 0.3566941022872925, -0.7320011854171753, 0.39859291911125183, 0.8338048458099365, -0.34885331988334656, -0.22338831424713135, -0.040411576628685, 2.354116678237915, -0.13221020996570587, -1.7909711599349976, 0.9939929246902466, -0.4421074092388153, -1.863772988319397, 0.6390257477760315, 0.372078001499176, 0.09067989885807037, -0.3413582444190979, -1.8003407716751099, 0.2497854232788086, -0.09198721498250961, -1.6167372465133667, -0.5353022813796997, -1.2009738683700562, -1.3447474241256714, -0.5067072510719299, 1.0644696950912476, -0.30521270632743835, 0.06831859797239304, -1.8480690717697144, 1.7644410133361816, 0.15802377462387085, 0.21121160686016083, -0.25156980752944946, -0.03367907926440239, 0.1335606873035431, 0.31092920899391174, 1.0115066766738892, 0.7935584187507629, 0.4385035037994385, 0.516726553440094, 2.7586371898651123, 0.6874586343765259, 0.30090200901031494, -0.28592321276664734, -1.737239956855774, 1.2696831226348877, -1.1544678211212158, -0.3310231566429138, -0.5723614692687988, 0.5935766100883484, -0.13701894879341125, 1.298587441444397], [1.0141785144805908, 0.2787356376647949, 0.5234174728393555, -0.9516198039054871, 0.533342719078064, 0.14799357950687408, -0.4883797764778137, -2.3833799362182617, 0.2314143031835556, -0.9690670371055603, -1.1491577625274658, -0.77333664894104, -0.20374104380607605, 2.109168291091919, -0.3168613314628601, 0.9081152081489563, -0.7126338481903076, -0.4184395968914032, 0.4457177221775055, -1.06803560256958, -0.5594552755355835, 0.6817177534103394, 2.9613921642303467, -0.7211971879005432, -1.0497087240219116, 0.8461174368858337, 1.4062148332595825, -1.0280896425247192, 0.1344779133796692, 0.22752277553081512, 1.8590692281723022, 0.990121603012085, -0.19428937137126923, 0.18271076679229736, 0.4901944696903229, -0.5091000199317932, 0.8898229598999023, -0.664966881275177, 0.9726380705833435, 0.28089192509651184, -0.4317418038845062, 0.5630913376808167, 0.34261757135391235, 1.7892850637435913, 1.5387741327285767, -0.6043345332145691, -1.0667792558670044, -1.6058191061019897, 0.39213404059410095, -1.7242720127105713, -1.162481665611267, 0.41126981377601624, 2.161736249923706, 0.6517265439033508, -0.5247413516044617, -1.2915877103805542, 0.320585161447525, -0.6063783168792725, -0.5031176805496216, 1.399290680885315, -0.12036089599132538, 0.865421712398529, -0.04858551174402237, -0.3918607532978058, 0.5860113501548767, 1.5402458906173706, -1.6392422914505005, -0.4553600251674652, -1.2230215072631836, 1.4597692489624023, 1.5239932537078857, 0.6539998054504395, 0.48196446895599365, 0.6748185157775879, -0.31831085681915283, 0.5381873250007629, 0.39566904306411743, -2.101543664932251, -0.1940605193376541, 0.16615454852581024, -1.066212773323059, -0.28157082200050354, 0.19321618974208832, -0.924919068813324, 0.29884424805641174, -0.5573627948760986, -0.1989843100309372, 1.3357704877853394, -1.1214873790740967, -0.5228282809257507, -0.4813956916332245, 0.697656512260437, -0.06342082470655441, 1.7162622213363647, -0.07758072018623352, 0.4413641393184662, 0.37365564703941345, 0.5417723059654236, -1.2752434015274048, 1.3914977312088013, 1.4225118160247803, -2.003232479095459, 1.6168816089630127, 0.574565052986145, 1.2611204385757446, -1.0410748720169067, -0.7717952132225037, -0.048692092299461365, -0.2164325714111328, 0.49355876445770264, -0.5267854332923889, -0.10942982137203217, 1.2463160753250122, -1.477711796760559, 0.7086997628211975, 0.2156875729560852, 1.0004689693450928, 0.8106789588928223, 0.38990819454193115, 0.3499144911766052, -0.1526554822921753, -0.9644506573677063, 2.4904863834381104, 0.7154467701911926, 0.21346713602542877, -0.4163859188556671, -0.15031099319458008, 0.2965020537376404], [-0.5773682594299316, -2.65643310546875, -0.6503128409385681, 1.1952029466629028, -0.751301109790802, -0.6199477910995483, -0.7404523491859436, -0.7617596387863159, 0.9394859075546265, -0.15530799329280853, -0.002965510357171297, -0.6641995310783386, 1.4652396440505981, -1.0815187692642212, -1.2729700803756714, 2.6432909965515137, 0.1257120817899704, 0.4059220254421234, 1.843742847442627, 0.17304295301437378, 0.9515150785446167, 0.4659690260887146, -1.3465440273284912, -1.0475285053253174, -1.1035641431808472, 0.9730293154716492, 0.4451615810394287, 1.6345088481903076, 0.20750795304775238, -0.4459620714187622, 0.20537284016609192, -0.9691748023033142, -0.9403908252716064, 0.6864529252052307, -0.5674440264701843, -0.8354660868644714, 0.5238839983940125, 0.8751201033592224, -0.22180229425430298, 0.3981810510158539, 0.42276257276535034, -1.7439817190170288, -0.1665111780166626, -0.1513587236404419, -1.077311396598816, 1.463586688041687, 0.2097272425889969, 2.501906156539917, -0.331900030374527, -0.4597042202949524, -0.7195814847946167, 1.1068884134292603, 0.887180745601654, -1.2013887166976929, 0.261190801858902, 0.5171136260032654, -0.06029545143246651, 0.8608894944190979, -2.840999126434326, 0.6582090258598328, -0.22182327508926392, -0.218641996383667, -1.0281931161880493, -0.5967628955841064, 0.4307977259159088, 0.7444051504135132, 1.5170140266418457, 0.09729678928852081, -0.8569130897521973, 0.7917195558547974, 0.3431234061717987, 0.5144625902175903, 1.8546786308288574, 1.0726442337036133, -1.082294225692749, -0.3388760983943939, -1.4648804664611816, -1.0785995721817017, -0.6137170195579529, 0.8901976346969604, 1.3949495553970337, 1.4139435291290283, -1.4066777229309082, 0.4255523085594177, -0.9777214527130127, -0.3547845482826233, 0.9616885781288147, 1.1528735160827637, 1.4386707544326782, -0.6459298729896545, 0.14588718116283417, -0.8591387271881104, -0.6852079629898071, -1.1919842958450317, 0.2071334570646286, -0.8147879838943481, -0.4943852722644806, 0.6518956422805786, -0.4077717065811157, -1.1327778100967407, -0.47576704621315, 0.7954860329627991, 0.08262189477682114, 0.6508235335350037, 0.3350944519042969, 0.3411354720592499, -0.2484237551689148, -0.9950767159461975, 1.5196539163589478, 2.213515043258667, 0.8288090825080872, 1.7501837015151978, -0.5170435905456543, 1.3725051879882812, -1.5215637683868408, -0.406000554561615, 0.2599843144416809, 0.09801970422267914, -1.195818543434143, 1.1106661558151245, -0.012055573053658009, -0.8910885453224182, 0.6860653758049011, -2.58782696723938, 1.9072877168655396, -1.527959942817688, 0.43956705927848816, 0.7910876870155334], [2.6190073490142822, -0.6345551013946533, 0.15654881298542023, 0.922415018081665, -0.9567162990570068, 1.2958672046661377, -0.206191286444664, -0.9781954884529114, -0.7321685552597046, 0.7007618546485901, 0.2650130093097687, 0.9841886162757874, -0.6976900100708008, 0.7477232813835144, 1.0279879570007324, 0.8159054517745972, 0.8773148059844971, 1.3243699073791504, -1.1803303956985474, 0.022058960050344467, -0.9551184177398682, 0.8229681849479675, 0.6485821604728699, -0.7579406499862671, -1.0899486541748047, -0.268798291683197, -0.7103654146194458, -0.2561142146587372, 1.71177339553833, 1.4681081771850586, 1.3059868812561035, 0.32699891924858093, -0.3882179856300354, -0.5969104170799255, -0.13271042704582214, 0.09402555227279663, 0.19680626690387726, -1.5222115516662598, 1.0000815391540527, 0.6550776958465576, 1.916835904121399, 0.7101430296897888, -0.3790178894996643, -0.3042687773704529, 0.8932454586029053, -1.2662856578826904, -0.9067181348800659, 1.4080584049224854, -1.4324244260787964, 0.7922170758247375, 1.4961482286453247, 1.1896401643753052, -1.2635178565979004, -0.05226491764187813, 0.5525719523429871, -1.04067063331604, -0.9263971447944641, -0.6295642852783203, 0.024678636342287064, -0.7096270322799683, -0.4618966281414032, 0.2632853388786316, -0.9738096594810486, -0.36775335669517517, -0.08415339887142181, -0.40714165568351746, 0.2768445611000061, 1.1314373016357422, -0.21371987462043762, 0.3669006824493408, -0.2084420621395111, 1.4437899589538574, -0.7148658037185669, 0.2934757471084595, 0.4946877062320709, -0.274130254983902, -0.5838860869407654, 0.5480419993400574, -0.29088833928108215, 0.8654425740242004, 0.46720901131629944, 0.8969773054122925, -1.0112656354904175, 1.4959264993667603, -1.2581653594970703, 0.5900666117668152, 0.7773799896240234, -1.4477660655975342, -0.47484204173088074, -2.8962388038635254, -0.8327948451042175, -1.0361100435256958, 0.7838690876960754, -0.7907845973968506, 0.33273911476135254, -0.20115713775157928, -0.26952627301216125, 0.9657031893730164, 0.9203658699989319, -0.296775758266449, 0.3130238652229309, -0.3106272518634796, -0.513761043548584, -1.627547264099121, 0.670407235622406, -1.0103733539581299, -1.2799599170684814, 0.43013995885849, 0.8859147429466248, 0.4393448233604431, -1.580958604812622, -1.1154699325561523, -0.7988765239715576, -0.2745530605316162, -1.3556857109069824, 0.2273760586977005, 0.20681911706924438, 1.595005750656128, 0.3931671977043152, -0.6078600883483887, 1.6358528137207031, -0.06837828457355499, 0.04897712171077728, 2.1959869861602783, 2.2080490589141846, 0.32007384300231934, 0.2296435385942459, -0.4343728721141815], [0.10660973191261292, 0.20620012283325195, -1.233215093612671, -0.1302013099193573, 1.2637335062026978, 0.18380682170391083, -0.6394187808036804, -0.3316882848739624, -0.6234696507453918, 0.1347484290599823, 2.055614709854126, 0.24959509074687958, -0.3525085747241974, -0.3493722081184387, 0.16172273457050323, 0.24206672608852386, 0.7415526509284973, -0.5489122271537781, -1.6071077585220337, 1.9382423162460327, 0.10216047614812851, 0.2723945379257202, -0.4680514335632324, -1.7000232934951782, -0.3049066662788391, -0.3164631724357605, -0.42860254645347595, 0.6193745136260986, -1.3574233055114746, 0.1679830104112625, -0.7340013384819031, -0.7819495797157288, -0.7485335469245911, 0.3154226839542389, -2.031698703765869, -0.17389875650405884, 0.924102783203125, 0.3554118871688843, 0.2643640339374542, -0.7782427668571472, -0.682273805141449, -0.14106062054634094, -0.21886493265628815, 0.6995849609375, -0.014027396216988564, 1.3579065799713135, 0.00012810285261366516, -0.9771595597267151, -0.26404815912246704, 1.5756914615631104, 1.3666293621063232, -0.048949986696243286, -1.225791335105896, -1.7706397771835327, 0.33328261971473694, -1.0166499614715576, 1.167086124420166, -0.2323908805847168, 0.12028958648443222, 0.39707323908805847, 0.1669100821018219, 1.0314463376998901, 1.472501277923584, -0.30509284138679504, 0.6810784339904785, -1.0480551719665527, -1.2050987482070923, 0.25020965933799744, 0.7350504994392395, 0.703513503074646, -1.9035753011703491, 0.5846598148345947, -0.030452866107225418, 0.36804044246673584, 0.4039076268672943, 0.3248462975025177, -0.299143522977829, -1.2462011575698853, 0.672847330570221, 0.2167336344718933, 0.010553599335253239, 0.05714103952050209, -1.468438982963562, 0.11427493393421173, 1.2439990043640137, -1.2744526863098145, -0.6570391654968262, 1.4090971946716309, 0.7306652069091797, -2.130199670791626, -0.1574118584394455, 1.0264228582382202, -0.923440158367157, -2.1052746772766113, 1.1238148212432861, -1.5338870286941528, -0.913249671459198, 0.38110464811325073, -0.5671474933624268, -1.927554965019226, -0.13884177803993225, 1.0471547842025757, -0.2089315950870514, 0.4571596384048462, 0.9952155947685242, 2.148427963256836, -0.5572170615196228, -1.5287835597991943, 1.4036736488342285, -0.3620815575122833, -2.600881576538086, 0.17947173118591309, 0.47111427783966064, -1.214441180229187, 1.0333019495010376, 1.2356728315353394, 0.9033457636833191, 0.785673201084137, 0.8408862948417664, -0.005804033949971199, 0.5783060193061829, 0.39478591084480286, -0.7601401805877686, 0.04594140127301216, -0.8249189257621765, 0.03520258888602257, -1.0560401678085327, -0.10986831039190292], [-1.1801735162734985, 1.3862175941467285, -2.3329389095306396, 0.7026486396789551, 0.6470919251441956, -1.6296234130859375, -1.3636178970336914, 0.8192855715751648, -0.06802505999803543, 0.2702970802783966, -0.5901293754577637, -1.3637357950210571, -1.7642707824707031, -1.1698195934295654, -0.4364323914051056, 0.10011555254459381, 0.23045074939727783, 0.18118810653686523, 1.4653433561325073, 2.1835334300994873, -1.0576698780059814, 0.04850425198674202, 0.926899254322052, 0.26515161991119385, 0.3048958480358124, 1.4868000745773315, 0.018100667744874954, 1.9353545904159546, -1.1087255477905273, -1.1541825532913208, 0.678111732006073, -0.5210803747177124, -1.018410563468933, 0.08152899891138077, -0.2600734829902649, -0.0992702767252922, -0.5651049613952637, 0.6413083672523499, 0.4133710265159607, 0.28467419743537903, 0.4852955937385559, 0.3486042022705078, 0.6368513107299805, 0.14856494963169098, -2.2078235149383545, 0.5144214034080505, 1.7596970796585083, -3.0397238731384277, -0.35413962602615356, -0.2796511948108673, -0.982568085193634, 1.4254803657531738, -0.4388135075569153, 0.6242180466651917, -0.6227752566337585, 0.6310142278671265, 0.0805584043264389, -0.8545986413955688, -2.0797057151794434, 0.15642452239990234, -0.2322118580341339, 1.5944883823394775, 0.5369894504547119, 0.24786871671676636, 0.09833991527557373, -0.8531870245933533, 0.926268994808197, 0.7687948942184448, 1.4340791702270508, 2.6096527576446533, 0.5160561800003052, -0.27964481711387634, 0.16903150081634521, 0.2865994870662689, -0.9641364216804504, -2.018389940261841, 1.0985503196716309, 0.012644958682358265, -0.9470762610435486, -0.9035395979881287, -0.10257662832736969, 0.8667472004890442, -0.4372125267982483, -0.8087064623832703, 0.20981548726558685, 0.4316079318523407, -0.2413109689950943, -0.3865751624107361, 0.165854811668396, -1.3651388883590698, -0.06544531881809235, -0.7799137234687805, -1.5902775526046753, -0.9249698519706726, -0.21242350339889526, 0.4019272029399872, 1.7423503398895264, -0.9646853804588318, -0.6301887035369873, -0.3715992867946625, -1.1469392776489258, -0.08131330460309982, -0.21142180263996124, 0.248740091919899, -0.4637015163898468, 0.2924555242061615, -0.5539728999137878, 0.5495312809944153, 0.9333049654960632, 1.043796181678772, -1.7258473634719849, 0.6918318867683411, -0.6675514578819275, -0.3630768358707428, -1.588450312614441, 2.3255531787872314, 0.5291022658348083, 0.9481614828109741, -1.373457908630371, -3.2185449600219727, 0.713922917842865, 1.147174596786499, 0.3087593913078308, 2.580979108810425, 0.2691936492919922, -0.6284129023551941, -1.6456528902053833, 0.848137378692627]]], [[[1.6114681959152222, 0.017052846029400826, 0.19572986662387848, -0.19501784443855286, 0.15694938600063324, -0.6211462616920471, 0.8278064727783203, 1.9289345741271973, 0.4122236371040344, 0.2257000058889389, 1.8826497793197632, 2.769172430038452, -0.3836764395236969, -0.28005391359329224, -0.17462223768234253, -0.19661317765712738, 0.9810340404510498, -0.2377920150756836, 0.6173184514045715, 0.008003205060958862, 0.5767202973365784, 0.945337176322937, 1.944865345954895, 0.4013625681400299, 0.8086712956428528, -0.0700070783495903, -1.216641902923584, 0.7471921443939209, -0.8573452234268188, -0.4579378068447113, 0.04889954999089241, -0.6714398860931396, -0.61138916015625, -0.8170205950737, 0.09132295846939087, 0.19691237807273865, -0.22490482032299042, 1.674835443496704, -1.1250187158584595, 0.018107516691088676, 0.6025002002716064, -1.2387515306472778, 0.9387080669403076, 2.700446367263794, -1.4459558725357056, 1.1484291553497314, -0.23627305030822754, 1.060265302658081, 0.02947433479130268, -1.7578167915344238, -1.519799828529358, -1.122679591178894, 1.2463430166244507, -1.9079056978225708, -1.063968300819397, 2.1372246742248535, -0.10600455850362778, -0.8237583041191101, -1.7671421766281128, 1.6738693714141846, -1.6997061967849731, 0.24724525213241577, -0.08725214004516602, 0.6049816608428955, -1.122886061668396, -0.5358344316482544, -0.9523624181747437, 0.4403078854084015, -0.641772449016571, -0.26138046383857727, -1.758357048034668, 0.5912069082260132, 1.7765640020370483, -1.2876752614974976, 0.5906689763069153, 1.0466551780700684, 2.237147331237793, -0.4903276264667511, 1.3338375091552734, -1.7666791677474976, -0.5179933309555054, -0.2526165544986725, -0.25872841477394104, 0.2996613085269928, -0.8705029487609863, 1.0287646055221558, 0.7279226183891296, -1.2605986595153809, 1.092078447341919, -0.7964134812355042, -0.8699178099632263, -0.37879815697669983, -0.4812411069869995, 0.27043986320495605, -1.3756165504455566, -0.9971351027488708, -0.5742130875587463, -0.3422462046146393, -1.1928054094314575, -0.1358419805765152, 0.5620903372764587, -1.183347463607788, -0.9072052836418152, -0.7213526368141174, -0.760939359664917, -0.7287012934684753, -0.432539165019989, 1.3094794750213623, 0.08996187150478363, 1.0470789670944214, 0.2605239450931549, -0.476816862821579, 0.37830114364624023, 1.2450066804885864, 0.5949891805648804, -1.3046691417694092, 1.186563491821289, 0.8540746569633484, -0.7239888906478882, -0.25802871584892273, -1.0930557250976562, -0.23984844982624054, -0.09445199370384216, 0.22363843023777008, -0.39376017451286316, 0.04576324298977852, 0.7835232019424438, -0.5521800518035889], [-0.7424989938735962, -0.03280823677778244, 0.2550150156021118, -0.13452588021755219, 1.1164746284484863, 0.3761998116970062, -1.3131829500198364, 2.050464391708374, -0.8758801221847534, -1.4162116050720215, -0.1744517982006073, 0.516800582408905, -0.13571231067180634, -0.27421098947525024, -1.6696834564208984, 0.45247748494148254, -1.1815334558486938, 0.7407426834106445, -0.3860825002193451, 0.3991314768791199, -0.6583446860313416, 2.17543888092041, -0.5870403051376343, -2.1604413986206055, 0.02968122437596321, 0.386599600315094, 0.5018931031227112, -0.7500529885292053, 0.8773338198661804, 1.4562404155731201, -1.1562559604644775, -0.3656889498233795, 0.11744938045740128, -1.7957631349563599, -1.203916311264038, -0.6373264789581299, -0.022461583837866783, 1.4279028177261353, -0.7758352160453796, -0.0949723869562149, 0.3802550733089447, 0.6543310284614563, -0.5720856785774231, -0.3966774046421051, -1.3496875762939453, -0.6062785387039185, -0.011124219745397568, 0.04442602023482323, -0.763307511806488, -1.2804988622665405, 0.001511493232101202, -0.1418483555316925, 1.090492844581604, -0.11281966418027878, 0.9822002649307251, 0.6303165555000305, 1.4483665227890015, -0.05436139553785324, -1.686078429222107, -0.1442566215991974, -0.39301609992980957, 1.0012749433517456, 0.6920636296272278, -1.0779105424880981, 0.4369531571865082, 1.0912737846374512, -0.9562809467315674, 1.382833480834961, -0.07593699544668198, 0.3840140104293823, 0.16687187552452087, 0.6876181960105896, 0.19558484852313995, 0.19464153051376343, -0.3841521143913269, -0.8067488670349121, 0.7797626256942749, 0.10037609189748764, 0.3123200833797455, 0.1872936487197876, 0.579750657081604, 2.7735073566436768, -1.7030030488967896, 1.0621070861816406, 1.5207176208496094, 0.8489383459091187, 0.6491948366165161, 1.1223284006118774, 1.7186241149902344, 1.6331747770309448, -0.32185426354408264, 0.9102970361709595, 1.933935523033142, -0.7911682724952698, -0.040190957486629486, -0.8820412158966064, -0.3964043855667114, -1.8400781154632568, 0.5415977239608765, 0.1969691962003708, -0.25004762411117554, 1.8797581195831299, -1.9710599184036255, 1.7724573612213135, -0.6467610001564026, 0.01892392337322235, 0.5620253682136536, 0.6530422568321228, -0.8118302822113037, -1.5661840438842773, 0.3305948078632355, -1.1358178853988647, 0.3784940838813782, 0.726852536201477, 1.5716052055358887, 0.4499979019165039, 0.14642666280269623, -0.07019921392202377, -0.8572326898574829, -1.2403188943862915, 1.6869351863861084, 0.025782795622944832, 1.385376214981079, 0.2919055223464966, -1.047010898590088, -1.171736478805542, 0.08743619173765182, -0.5453655123710632], [1.2743018865585327, -0.6665369868278503, -0.5602602958679199, -0.3900386095046997, 0.2588574290275574, -0.7890298366546631, 1.1060564517974854, -0.21698515117168427, -1.2553651332855225, 0.3743293881416321, -1.8903110027313232, 0.335135817527771, -0.0225206408649683, -1.6512012481689453, -1.3234790563583374, 0.16060954332351685, 0.26970455050468445, 1.6142185926437378, -0.636706531047821, -0.37285470962524414, 0.5795681476593018, 0.1943039894104004, 1.9368177652359009, 0.263140469789505, -1.3249973058700562, 1.029655933380127, -0.3182389736175537, 0.12967635691165924, 0.11666544526815414, -0.8070700168609619, -1.3684132099151611, 0.22205552458763123, 0.9709224104881287, 0.23465240001678467, 0.164557546377182, 1.3802179098129272, 3.2231709957122803, -0.745555579662323, 0.3337613344192505, -0.4845401346683502, -0.7245627641677856, -1.2206932306289673, -0.3605611324310303, -0.12645822763442993, -0.048865970224142075, 0.2865484356880188, -0.8041360378265381, -0.9989582300186157, 1.800734043121338, 0.9590627551078796, 0.8505973815917969, 0.9749557971954346, 0.5826209187507629, 0.22418175637722015, 0.4189585745334625, -1.6463465690612793, 2.389561891555786, -1.2826182842254639, 0.5008214712142944, -0.63294917345047, 0.14525394141674042, 1.3136365413665771, -0.5231408476829529, -0.20434948801994324, 0.3427428901195526, 0.939109206199646, 1.7799782752990723, -1.6921205520629883, -1.3387037515640259, -0.6246964931488037, 0.2725951373577118, 0.4406343698501587, 0.7713127136230469, -0.23573845624923706, -1.6239843368530273, -1.7547674179077148, -0.2460053563117981, 1.279697060585022, 0.7629135251045227, 1.6673160791397095, 0.08141572773456573, -1.7435652017593384, 0.06147952377796173, 0.24504677951335907, 1.9542217254638672, -0.41222187876701355, 0.39046502113342285, -1.1270949840545654, -1.789355993270874, -0.3896541893482208, 0.4690692126750946, -1.544447898864746, 1.028784155845642, 0.8899204730987549, 1.71079683303833, -0.15054509043693542, -0.116187684237957, 0.24425062537193298, 0.7210277318954468, 0.5662891864776611, 0.24059075117111206, 0.513857901096344, -0.17779965698719025, -0.7905305624008179, 1.6527667045593262, 0.5485336780548096, 0.687167227268219, 1.2128617763519287, -0.46671250462532043, -0.8007475733757019, 0.23254261910915375, -0.3478644788265228, 1.3322333097457886, -0.8127869367599487, 0.7672334313392639, 0.05638227239251137, 1.193785309791565, 0.1863238364458084, 0.8037464618682861, 0.21820266544818878, -0.37264904379844666, 0.7983027100563049, -1.388285517692566, 0.7252005934715271, -0.8985471725463867, -0.06332903355360031, -0.6868508458137512, 1.6392313241958618], [-0.6145817637443542, 1.3555777072906494, 0.5472904443740845, 0.8497557044029236, 0.9408813714981079, 0.4738786518573761, -0.09996115416288376, -0.7126553058624268, -0.5354340672492981, -1.271727204322815, 1.2036975622177124, 0.20633839070796967, 0.18180347979068756, 1.627047061920166, -1.0810948610305786, 1.120049238204956, 0.597825288772583, -0.3397917151451111, 1.697145938873291, 1.353358507156372, -1.3695794343948364, -0.8355123996734619, 0.4040326774120331, 0.8682169914245605, -0.5675647854804993, 1.8687794208526611, 1.3230372667312622, 1.6898401975631714, -0.18858467042446136, -0.001517912489362061, 0.7845788598060608, 2.46915602684021, -0.5984696745872498, -0.4503878951072693, 0.03182317316532135, -0.4218136668205261, -0.11403490602970123, -0.12836380302906036, 1.1473286151885986, 0.3593135178089142, 0.174742192029953, 0.11671391129493713, 0.8244228363037109, -0.4093784987926483, 0.8046610355377197, -0.7722054123878479, 0.039225392043590546, -0.5403211116790771, -2.242746114730835, -0.4900726079940796, -0.9171967506408691, 0.9719538688659668, -0.7582520246505737, 0.27040016651153564, -0.37790459394454956, -0.009396227076649666, -0.02144021913409233, 0.15266089141368866, -0.11368994414806366, -0.8779848217964172, -1.187319278717041, 0.6401523947715759, 1.2329115867614746, -1.9158854484558105, -0.6580235362052917, -1.4933292865753174, -0.2985228896141052, 1.6830214262008667, 0.32284533977508545, 0.39314475655555725, -1.2036631107330322, 0.5703000426292419, -1.176727294921875, -0.41437700390815735, -0.2540428936481476, 0.9067468643188477, -0.6032270193099976, -0.42552560567855835, -0.11689291149377823, -0.2646745443344116, 1.171585202217102, -0.6867073178291321, -1.2794299125671387, -1.0701634883880615, 1.1136597394943237, -0.4899933636188507, 1.4169843196868896, -0.2736440598964691, 1.172632098197937, 0.7168624401092529, -0.5265550017356873, 0.03944740816950798, -0.6064251065254211, -0.017217321321368217, -0.7872809767723083, -0.18357089161872864, -0.3748144209384918, -0.07091172784566879, -1.7736159563064575, -0.380434513092041, 0.015327059663832188, 0.7428128123283386, -0.6984685063362122, 1.22931706905365, 0.9793538451194763, 0.3413407504558563, 2.019838809967041, 0.7336992025375366, 1.784353494644165, 0.04262779280543327, -1.440950632095337, -0.6440224051475525, 0.19987109303474426, -0.2387741208076477, 0.2942734360694885, 0.811983048915863, -1.2367022037506104, -1.6815613508224487, 0.011331523768603802, 0.5510686635971069, -0.10045895725488663, -1.3562229871749878, -0.3276311457157135, 0.373207688331604, -0.7047067284584045, 0.8226242661476135, 0.5751841068267822, 0.05509961396455765], [-0.4641976058483124, -1.9544650316238403, 1.308289647102356, 0.714760422706604, -0.3306088149547577, -1.3673251867294312, 1.3893722295761108, -0.08521683514118195, -1.1733150482177734, -0.004216213710606098, 0.3463296890258789, -2.546294927597046, 0.3104054033756256, -1.0794519186019897, 0.6850566267967224, 0.25559964776039124, -2.7900660037994385, -0.3886893093585968, -0.9205626249313354, 0.268241286277771, -0.4844031035900116, -1.2412093877792358, -0.08284082263708115, -0.23874260485172272, 0.3171366751194, 0.36890703439712524, 0.5974087119102478, -0.5044571161270142, 0.0008429450099356472, -0.19444945454597473, 0.5011541247367859, -0.7260092496871948, 0.19103214144706726, -0.4138515889644623, -0.019316773861646652, -0.30879032611846924, 0.7592621445655823, -0.6275843381881714, -1.2355376482009888, -1.2922489643096924, -0.31654393672943115, 0.10112912952899933, 0.500247061252594, 0.5418460369110107, 1.0419135093688965, -0.5263984203338623, -1.281078815460205, 0.15007582306861877, 0.32104453444480896, -0.4613300561904907, -0.152603417634964, -0.0786171481013298, -0.7631554007530212, -0.20214688777923584, 0.26099053025245667, 1.7782409191131592, 0.1448216289281845, -0.7415169477462769, 0.2761535048484802, 1.0625648498535156, -0.5714046359062195, -0.10141769051551819, -1.0817229747772217, -1.1409803628921509, 0.17419832944869995, -0.5504512786865234, 0.723211407661438, -0.3602272868156433, 0.26990750432014465, 0.006988361477851868, -0.817601203918457, 0.9313555955886841, -0.3251747488975525, -1.4481754302978516, 0.03550131991505623, -1.121788501739502, -0.6166446208953857, -0.44710883498191833, -0.0910884290933609, -0.3954557776451111, -0.26208850741386414, -0.3169926702976227, -0.060150980949401855, 0.2941366136074066, -1.0710313320159912, 0.28854280710220337, -1.3362245559692383, -1.750163197517395, 0.8028557300567627, 0.5910372734069824, 0.8363203406333923, 0.3728267252445221, 0.9148370027542114, -0.5696233510971069, 0.7566992044448853, -0.5219119787216187, -0.30980199575424194, 0.01782647892832756, 0.14617742598056793, 1.2091925144195557, -0.6440638899803162, -0.036973707377910614, -1.3953291177749634, 0.785209596157074, -1.6438945531845093, 0.4638725519180298, 0.1107397973537445, -0.2009548842906952, -1.2855103015899658, 1.5015202760696411, -1.6559499502182007, 0.5854983925819397, 0.5454532504081726, -1.2871953248977661, 1.5932621955871582, 1.4965256452560425, 0.5197780132293701, -0.49370676279067993, -0.21280993521213531, 0.09300868958234787, -1.2357614040374756, -0.6482335329055786, 0.7104005217552185, -0.47818464040756226, -0.24550117552280426, 0.39796119928359985, -1.1993237733840942, -0.12500078976154327], [1.2311959266662598, 0.10258942097425461, 1.9555261135101318, 0.10709680616855621, 1.466759204864502, 0.42832475900650024, 1.2266316413879395, 0.6974170207977295, 1.5264966487884521, 0.06667595356702805, 2.5413527488708496, 1.0806635618209839, -1.3583790063858032, 0.8612271547317505, -1.1193506717681885, -0.797844409942627, 1.2324713468551636, 0.907992422580719, 0.7744592428207397, -0.3052336871623993, 0.09758996963500977, -0.17903968691825867, 0.48030635714530945, -0.05159677565097809, 1.7630785703659058, -1.36818528175354, -1.0275771617889404, 0.3061610758304596, 0.6594834923744202, 1.8625339269638062, -0.12147165089845657, 0.8120130300521851, 0.41412660479545593, -0.4359879493713379, 0.41357025504112244, 0.34610310196876526, -1.3199565410614014, 1.6200618743896484, 0.5587107539176941, 0.043417125940322876, -2.018272876739502, 0.68642657995224, -1.0088610649108887, 1.4637823104858398, 2.0455150604248047, 0.6261509656906128, -0.09882772713899612, -0.5916904211044312, 0.958097517490387, 0.6941748261451721, -0.859345018863678, -3.5779478549957275, -1.6839022636413574, -0.8678983449935913, 2.4152071475982666, 0.45713263750076294, 0.8836647868156433, 1.364461064338684, -2.7994120121002197, -1.7043979167938232, -0.49518123269081116, -2.575826406478882, 0.34999436140060425, -0.14439493417739868, 0.394258588552475, -0.4600594937801361, 0.2240164428949356, -0.6549811959266663, 0.5280929803848267, -1.6988110542297363, 0.11155535280704498, -0.3057330548763275, -0.18356554210186005, -0.6429003477096558, -1.6315951347351074, -0.2826589047908783, 0.21879024803638458, -0.07542135566473007, -0.4077405035495758, -0.2232557088136673, -1.6143016815185547, 0.4394330382347107, -0.2901459038257599, -0.05315065011382103, -1.2537287473678589, 1.2957419157028198, 1.1418343782424927, 0.8978950381278992, 1.727258324623108, 0.3194597363471985, -1.0112770795822144, 1.7103016376495361, -0.2627737522125244, -0.04685678333044052, -0.5937942266464233, 0.20933493971824646, -0.0549761988222599, 1.5429437160491943, 0.20283980667591095, -0.7130616903305054, -0.6950035095214844, 0.533301591873169, 1.5597370862960815, 1.2166333198547363, 0.0773351788520813, -0.2418670654296875, 1.1803526878356934, 1.8388644456863403, -0.47247788310050964, 0.8442955613136292, 0.5365671515464783, -1.073071002960205, 0.8060349822044373, -0.4845980405807495, -0.5442185401916504, 0.26089218258857727, -0.8720182180404663, -0.9131919741630554, -1.2459242343902588, 1.3368306159973145, -0.632648229598999, -0.8671178221702576, -1.2222981452941895, -1.0834490060806274, 0.5501255393028259, -0.05613720789551735, 0.22958461940288544, -1.2429914474487305], [-1.714491844177246, -0.6347303986549377, -0.2946856617927551, 0.2379903346300125, -1.4734185934066772, 0.29790741205215454, -0.07802501320838928, -0.2360563576221466, 0.7414060235023499, -0.9531006217002869, -0.5805749297142029, 0.30436408519744873, 1.4732520580291748, -0.33137810230255127, -1.2863785028457642, -0.31028926372528076, 1.4246177673339844, 0.5767838358879089, 0.10604143142700195, -0.3439427614212036, -0.4063183665275574, -0.0849519744515419, 0.9373968839645386, 0.684296727180481, 0.42916205525398254, -0.2737266719341278, 0.3294467628002167, 0.8671206831932068, 1.3817540407180786, 0.2660759389400482, 1.7960619926452637, -1.2898526191711426, 1.1206938028335571, -0.4757210910320282, 1.472503900527954, -0.9008206725120544, -0.6474352478981018, 0.03427322581410408, 0.5251424908638, -0.22707059979438782, -2.20086932182312, -1.2638698816299438, -0.562657356262207, -0.2481290102005005, -0.37810030579566956, 1.5825204849243164, -0.8153364658355713, 1.9443891048431396, 0.6693671345710754, 1.2303662300109863, -0.8275127410888672, 0.3901842534542084, -0.28901541233062744, -0.1300983428955078, -0.9806925654411316, -0.22174179553985596, -1.8487530946731567, 0.8840495944023132, 0.12154818326234818, -1.4990981817245483, 0.4454886317253113, 0.48168739676475525, -2.140080451965332, 0.19807294011116028, 0.25157293677330017, 0.7076099514961243, -1.139668345451355, -0.6172568202018738, -0.08163099735975266, -0.8766106367111206, 0.3059670627117157, 0.1993335783481598, 0.19196172058582306, -0.9191423654556274, -0.16477468609809875, -3.215348958969116, -2.1353838443756104, 0.0031970820855349302, 0.873222827911377, 1.1200529336929321, 2.55513858795166, -0.15162451565265656, 1.1250567436218262, 0.45939821004867554, 0.7612545490264893, 0.9868351817131042, -1.263116717338562, -0.46192073822021484, 1.0799566507339478, 0.41298285126686096, 1.3050235509872437, 1.3207001686096191, -0.05025516450405121, -0.8641595840454102, 1.144120693206787, 1.564333200454712, 0.19433364272117615, 0.6728414297103882, 0.6216427683830261, -0.19280025362968445, -2.1635096073150635, 1.5330970287322998, -0.17075186967849731, 1.7859426736831665, -0.4843713939189911, 0.6727879047393799, -0.6604363322257996, 0.5925912857055664, -0.010641864500939846, 1.2809715270996094, 1.0944839715957642, -1.9114758968353271, 0.5747637152671814, 1.2247223854064941, 0.39688706398010254, 1.8958873748779297, -0.9960854649543762, -0.42001280188560486, 0.08745576441287994, -0.3034358620643616, 0.5302267074584961, 0.7348001003265381, 1.8926928043365479, -0.4247939884662628, 1.2061570882797241, -0.1511940062046051, -0.5581222176551819, -0.6934301257133484], [0.7474884390830994, -0.06799327582120895, -0.910951554775238, 0.12469357997179031, 0.4046706259250641, -0.04749695584177971, 1.2398972511291504, 1.2488263845443726, 1.1875747442245483, 0.6272354125976562, -0.7788571715354919, -0.05833977833390236, -0.3236086964607239, 0.384835809469223, 0.08875381201505661, 0.380257785320282, -0.5718256235122681, -0.3116091787815094, 1.1651846170425415, -0.605612576007843, 0.7840871214866638, 0.25175800919532776, -0.6591817736625671, 0.8964822292327881, 1.1405686140060425, 0.34989774227142334, 0.9669845700263977, -0.9241471886634827, -0.19147421419620514, 1.46219801902771, -1.950956106185913, 0.4560844600200653, -0.5753825902938843, -0.6006619930267334, 1.7238250970840454, 0.3765130937099457, -0.7904734015464783, 0.3790058195590973, 1.1361697912216187, 0.598193347454071, -2.912952184677124, 0.580157995223999, -0.9844574928283691, -0.46105432510375977, -0.007509063929319382, 0.07256154716014862, 1.964103102684021, -0.39126890897750854, 0.6412227153778076, -0.14412236213684082, -2.6235992908477783, 1.2945747375488281, -0.5856459736824036, 0.049765344709157944, -0.1465476006269455, -0.22872400283813477, 0.004428073298186064, -0.022599587216973305, -0.2856747508049011, 0.7193741798400879, 0.2836421728134155, -0.19222936034202576, 0.8623191118240356, 1.838975191116333, 2.4817631244659424, 0.8693186640739441, -0.21272961795330048, -0.869084358215332, 1.8465344905853271, -0.4997314512729645, -0.9045038223266602, 0.2383774220943451, 1.0667885541915894, -1.1526200771331787, 1.066538691520691, 1.0844805240631104, 0.7286785244941711, 0.647631824016571, 3.2142419815063477, -0.4805781841278076, 0.43355706334114075, 0.1781807541847229, -0.2406168431043625, -0.910740852355957, 1.0841351747512817, -1.3441475629806519, 0.13226310908794403, -0.713828980922699, -0.44314372539520264, 0.3014953136444092, -1.622758388519287, -0.8944475054740906, 0.6331118941307068, -1.8196829557418823, 0.6497189402580261, -1.776561975479126, -1.6088111400604248, 0.332492470741272, -0.9378533363342285, 0.6769752502441406, -1.5195125341415405, -1.3426507711410522, 2.117830753326416, -1.1269571781158447, 0.44817623496055603, -0.28272485733032227, 1.3324118852615356, -1.846662163734436, -0.6796501278877258, 0.060445018112659454, 0.3306014835834503, -0.4188143014907837, 0.519186794757843, 0.31491756439208984, 0.2925812900066376, 0.5220747590065002, -1.9658797979354858, -0.3730635643005371, -1.3364940881729126, -0.45473572611808777, -0.20460687577724457, 0.7583228945732117, 0.6615340113639832, -1.0734398365020752, 0.6154493093490601, 0.3571394681930542, 1.4694606065750122, -1.3716716766357422], [-0.7258346676826477, 0.6503542065620422, -1.1291064023971558, 0.5042828321456909, -1.728350043296814, -2.5117764472961426, -0.08450772613286972, 0.6054664850234985, 1.3921856880187988, 0.3729904890060425, 0.9109609127044678, 0.6931026577949524, 1.5785027742385864, -1.8831239938735962, 0.8246064782142639, -0.6322450637817383, 1.3085224628448486, 0.08957745134830475, -2.140742778778076, 0.30535778403282166, 0.2915051579475403, -0.19063779711723328, 1.502825379371643, 3.1082518100738525, 0.41951730847358704, -1.2568621635437012, -0.6412025094032288, 1.0541555881500244, -0.6108108162879944, 1.5000561475753784, -0.2084624320268631, -0.6059306263923645, -0.7370836734771729, -0.6130867004394531, 0.8136938214302063, -0.5603231191635132, -2.5801637172698975, 0.2692163288593292, -1.1320785284042358, -1.4437838792800903, -1.3316699266433716, -0.24320584535598755, 0.020125219598412514, 1.31147038936615, 1.0033525228500366, -0.028189877048134804, -1.03609037399292, 1.5828925371170044, -1.0016770362854004, -0.18849416077136993, -0.8130632638931274, -0.43895527720451355, -1.4109704494476318, -0.9390808939933777, 2.296294927597046, 0.6091393828392029, 0.16247870028018951, -0.6817026734352112, -0.8560538291931152, -0.8295933604240417, -0.44562044739723206, 0.10867951065301895, 0.1222921833395958, -0.4076969623565674, 1.0222132205963135, 0.9814363121986389, 0.665877103805542, 1.3471059799194336, 1.0144861936569214, -0.8578988313674927, -1.515700101852417, -0.2703171670436859, -0.18242597579956055, 0.3164568543434143, -0.95777428150177, 2.898493528366089, -0.3782416582107544, -0.6386073231697083, 0.7680052518844604, 1.7969049215316772, -0.8614674210548401, -0.4605727195739746, 0.43978041410446167, 1.604691505432129, 1.6389930248260498, 1.213295817375183, 0.5965196490287781, -0.17588026821613312, -0.3358912169933319, -0.2397783398628235, -0.7426292300224304, 0.8212267160415649, 0.676091730594635, -0.9811277985572815, -0.1124112457036972, 0.5763735175132751, -0.36963582038879395, -0.212765634059906, 0.33617153763771057, -1.2930705547332764, -1.0421741008758545, 0.1853434294462204, -2.923377513885498, 0.1614043265581131, -0.4007115066051483, 0.777037501335144, -0.07326064258813858, 1.2743533849716187, -0.22958879172801971, -0.21278803050518036, 0.7270581126213074, -1.0044176578521729, 0.1377888321876526, -2.3172404766082764, -1.38308584690094, -0.4159771502017975, -0.389163076877594, 2.099414825439453, -2.668869733810425, 1.5583713054656982, 0.4277917444705963, 0.45841267704963684, 0.5638061165809631, 1.1178672313690186, 0.8675143122673035, -0.4348936676979065, -0.9670417308807373, -1.0257960557937622], [0.37197479605674744, -2.093449592590332, 2.207517385482788, -1.040743350982666, -0.9474741220474243, 0.8485539555549622, 0.353816956281662, -0.2982100546360016, -0.7253183126449585, 1.4487226009368896, -0.8348000049591064, 0.04134898632764816, 0.4252181649208069, -0.6585172414779663, 1.5554007291793823, 1.5529534816741943, -0.6225181221961975, -0.8524770140647888, -0.7415375709533691, -0.36850762367248535, -2.048504590988159, 1.7314512729644775, 0.062084004282951355, -0.6387932896614075, -0.28693604469299316, -0.3737780451774597, -0.5468543767929077, -1.6127010583877563, -3.4100077152252197, 1.477806806564331, 0.7254151105880737, 0.19878247380256653, -2.2793967723846436, -0.8749262690544128, -1.332904577255249, -1.2614593505859375, 0.604945957660675, 0.32444649934768677, 0.1520593762397766, 0.06262468546628952, 0.496349573135376, -1.1159945726394653, 0.7325518131256104, 0.5833197236061096, -0.6640627980232239, 2.7965095043182373, 0.7536365985870361, -0.521671712398529, -0.5052353739738464, 1.634560465812683, 1.5945780277252197, 0.5166317820549011, 1.4763519763946533, -1.002570390701294, 1.9229646921157837, -0.6931776404380798, -0.6040016412734985, 1.0820032358169556, -0.31905028223991394, 0.256123423576355, -0.3267075717449188, 1.2949572801589966, -1.1598148345947266, 0.3313535153865814, -0.13453075289726257, 1.246432900428772, 0.5725931525230408, -0.05471525713801384, 2.1238627433776855, -0.23037633299827576, 0.541146457195282, -2.2684481143951416, -0.8469464778900146, 0.305416077375412, 0.23677238821983337, -0.010323084890842438, 0.08930502831935883, 0.025715243071317673, 1.2059341669082642, 0.2703794538974762, 0.4706503748893738, 0.6604426503181458, -0.1795744001865387, -0.7075977921485901, -0.048669926822185516, -0.4656006693840027, -0.9617737531661987, -0.08498990535736084, 0.3289954960346222, -1.4867403507232666, 0.2595749497413635, -0.9144803881645203, -0.34002113342285156, 0.7532406449317932, 1.1024836301803589, 0.9825917482376099, 0.02256365306675434, -1.418349266052246, -0.47914308309555054, 1.2342250347137451, 0.5293542146682739, -0.6263045072555542, 1.6048481464385986, 0.41892457008361816, 3.3763909339904785, -0.5577920079231262, -0.06769789010286331, -0.9239026308059692, 0.39159145951271057, -0.7558256983757019, 0.46375080943107605, 0.7972218990325928, 0.45386263728141785, -0.9137935638427734, -0.6017060875892639, 0.19502834975719452, 0.6648034453392029, -0.7614090442657471, -0.23938210308551788, 1.6023929119110107, -0.6323022842407227, 0.5439485907554626, 0.3023745119571686, -0.282604843378067, 0.06087160483002663, -0.1803155541419983, -0.09619323164224625, -0.3590713441371918], [-1.90224289894104, -0.158871129155159, 0.3333779275417328, 0.02121693082153797, -2.512896776199341, -0.23192289471626282, -1.1685552597045898, 1.3103569746017456, -1.0653860569000244, -0.5476825833320618, 0.9706496596336365, 0.7100138068199158, -0.018021762371063232, 0.3547441363334656, -1.0885615348815918, 0.4170688986778259, 1.807108998298645, -0.22540068626403809, 0.4619079828262329, -0.2936898469924927, -1.513810157775879, 0.0598687008023262, 0.6517658233642578, -1.1491467952728271, 0.9519758820533752, -0.01164940930902958, -0.40305042266845703, -1.8479565382003784, -1.2652065753936768, 1.128728985786438, -1.2436842918395996, -0.27843090891838074, -0.07324931770563126, 0.4575690031051636, 0.9165898561477661, 1.5311036109924316, 2.2067182064056396, -0.7677581310272217, -1.4316296577453613, 1.9177284240722656, 0.5772014260292053, 1.314812183380127, -1.4316290616989136, 1.2212811708450317, -0.1826387643814087, 1.5130929946899414, -1.1924885511398315, -0.1607782244682312, 1.2097676992416382, -0.48406746983528137, -0.5433106422424316, -0.2031116634607315, -0.798462450504303, -1.9812288284301758, -0.09655655920505524, 0.4950880706310272, -1.9533655643463135, -1.1040409803390503, 1.5004185438156128, 1.156367301940918, 0.3464082181453705, -1.46290922164917, -1.429982304573059, 0.6107665300369263, -0.4049195945262909, -0.6638866066932678, -0.17054758965969086, 1.247018575668335, 0.9137541055679321, 0.8595224022865295, -0.2432529777288437, 0.018930526450276375, 0.25801336765289307, -0.09152347594499588, 0.8237398266792297, 0.5147963762283325, -0.7959043383598328, -0.9873747229576111, 0.5289035439491272, -0.6136803030967712, -0.3091985881328583, -0.21401305496692657, 0.62619948387146, 0.8559893369674683, -0.5514700412750244, -0.5025750994682312, 0.6896435618400574, -0.5070998668670654, -0.2817957401275635, 0.3606778681278229, 0.8667028546333313, 1.0586802959442139, -0.154384046792984, -0.4733697772026062, -1.0383564233779907, -0.9786238670349121, -2.0611884593963623, 0.7400951385498047, 0.2059272676706314, 0.1938478797674179, -0.5356141924858093, -0.25497376918792725, -0.7834951877593994, -0.5744054317474365, -2.5222666263580322, -0.8534173965454102, 0.07611039280891418, 0.3407120406627655, 2.3937859535217285, -0.8061663508415222, -0.33947548270225525, -0.041030559688806534, 0.5070490837097168, 0.1073099747300148, -0.09163623303174973, -1.9471441507339478, -0.96329665184021, 0.6397266387939453, 1.4699770212173462, 1.4486654996871948, -0.17374607920646667, 0.2938459515571594, -1.7652831077575684, 2.008971929550171, -1.794545292854309, 0.6826202869415283, 0.5198675990104675, 0.5496917963027954], [0.7242059707641602, 1.0664674043655396, -0.290727436542511, -0.327944815158844, 0.15730158984661102, -1.6774532794952393, -1.2758909463882446, -1.7725327014923096, -0.25686609745025635, -0.1246669813990593, 0.6164315342903137, 0.9507557153701782, -0.4944698214530945, -0.6617539525032043, -0.9633676409721375, -0.6658565402030945, 0.5015702247619629, 0.23295943439006805, -0.2292712777853012, 1.171357274055481, -1.2862341403961182, -1.162933111190796, -0.11489561200141907, 0.3049720823764801, 2.0498363971710205, -0.8768438696861267, -2.184422492980957, -0.3680953085422516, 0.32223090529441833, 0.07431994378566742, -1.0573526620864868, 1.2892496585845947, -0.7119978070259094, 1.7313339710235596, 0.28745827078819275, -1.4176489114761353, 1.3855558633804321, -0.3739689290523529, -0.16779738664627075, 0.7103153467178345, -1.7477120161056519, 0.43338558077812195, -0.9619821310043335, 0.5501086115837097, 0.5129369497299194, -0.30366250872612, -0.06046091020107269, -0.976003110408783, -1.382534146308899, 0.5125504732131958, 0.23971213400363922, 1.8768203258514404, 2.5692851543426514, -0.41036462783813477, -0.6577705144882202, 0.3747493028640747, 0.22880129516124725, -1.6163662672042847, 1.1730568408966064, 0.3856845796108246, -0.7270981073379517, 0.33624228835105896, 0.37471136450767517, 0.09096021950244904, -0.7988207340240479, -0.12746168673038483, -0.07829400897026062, -1.0659457445144653, -0.07102145999670029, -0.7654603719711304, -0.27471694350242615, 0.41457468271255493, 0.14080365002155304, 0.48445677757263184, -0.810109555721283, 0.2890060842037201, 0.7018107175827026, -0.3388465642929077, -0.820682168006897, 0.46082285046577454, -0.534784734249115, -2.094252586364746, -2.273460865020752, 1.1575020551681519, -0.32230299711227417, 0.70100998878479, 1.3851252794265747, 1.0308606624603271, -1.6027591228485107, -0.8259715437889099, 1.1695547103881836, 0.2427307814359665, 0.21760235726833344, -0.34668633341789246, -1.409441351890564, 0.3887992799282074, -1.9820562601089478, -1.3865628242492676, -0.4614228308200836, 0.2304258644580841, 1.3969693183898926, -0.10439911484718323, 1.2017261981964111, 0.45510637760162354, 0.019859574735164642, -1.0220370292663574, 0.07914157211780548, -0.24909579753875732, 1.2440606355667114, -0.6404922008514404, 0.5702099204063416, 0.012305361218750477, 0.4915119409561157, -0.6098846793174744, -1.0420973300933838, -1.2685190439224243, -1.4510469436645508, 1.203069806098938, 1.278320550918579, -0.5930480360984802, 1.5088422298431396, -0.6551601886749268, -0.6544999480247498, 0.5572515726089478, -1.5824434757232666, 0.626147985458374, -0.2168692797422409, -0.38498127460479736], [-0.28012794256210327, -0.8711268901824951, -0.6289159655570984, 2.621203660964966, -1.0112298727035522, 1.7113083600997925, 0.7631597518920898, -0.17752403020858765, -0.7169491052627563, -0.08914820849895477, -0.8330756425857544, -0.15214534103870392, -0.233809694647789, -1.712600827217102, -0.2413458675146103, 0.34060391783714294, -1.0503705739974976, -0.31738924980163574, -0.8288105130195618, -0.7138050198554993, 0.17779883742332458, 0.13412980735301971, -0.21206893026828766, 0.7020806670188904, 0.32442760467529297, 1.5668041706085205, -1.0498956441879272, 0.28056225180625916, 1.3377423286437988, 0.39101341366767883, 0.3272267282009125, 0.4407637417316437, -1.497257113456726, -0.5394710898399353, -0.20793525874614716, 1.2022899389266968, -0.14635825157165527, 0.3873312175273895, 0.2505775988101959, 1.547475814819336, 0.7240371108055115, -0.07216444611549377, 0.7447760105133057, 0.7533393502235413, -0.7019867300987244, 0.8725572228431702, -1.0539097785949707, 1.5318653583526611, 0.16424019634723663, -0.4595496654510498, 1.145179271697998, -0.33498427271842957, 0.3077842593193054, 0.7339674830436707, -1.34089195728302, -0.8880772590637207, 0.8624143004417419, 0.46525025367736816, 0.9293149709701538, 0.03752619028091431, 1.4381295442581177, 0.9170344471931458, 0.5356391668319702, 0.22801364958286285, 0.1038832888007164, -0.058615542948246, 0.48040586709976196, -0.24935266375541687, -0.6905471682548523, 1.1610491275787354, 2.2542567253112793, 0.8386175632476807, 0.6288226842880249, 0.5600385069847107, 1.0301473140716553, 0.6219842433929443, 0.6500443816184998, -2.439257860183716, 0.837981104850769, -0.6396063566207886, 0.35380664467811584, 0.6771210432052612, 0.6273419260978699, 0.3825659453868866, 1.1480234861373901, -0.024742646142840385, 0.40474754571914673, -0.82086181640625, -0.07794907689094543, 0.47488895058631897, -1.1006247997283936, -1.018669605255127, 1.757949948310852, 1.579319715499878, -0.9633601903915405, -0.415301114320755, 0.29113924503326416, -0.6230173110961914, -0.0866864025592804, 0.0776248574256897, -0.7770290970802307, 0.7234606742858887, 0.4202669858932495, 0.22055061161518097, 0.2591205835342407, -2.7950947284698486, 2.245898485183716, 1.1576473712921143, -0.6324850916862488, 0.1966911107301712, -1.2768621444702148, -1.0942504405975342, 0.2348089963197708, 0.9256511330604553, -0.2715585231781006, 0.037284303456544876, -0.9858050346374512, 0.7910052537918091, -0.22468797862529755, 1.1040695905685425, -0.35838615894317627, 1.5362694263458252, -0.65435391664505, 1.148768663406372, 0.874259889125824, 1.6122405529022217, -2.044886589050293, 0.9561627507209778], [-0.45741957426071167, -0.7902386784553528, -0.2378256618976593, -1.15081787109375, 0.38749921321868896, 0.061511535197496414, 1.1288288831710815, -0.23878665268421173, 0.6361862421035767, 0.12902414798736572, 0.43024858832359314, -1.1338914632797241, -0.529725193977356, 1.2826355695724487, -1.5631107091903687, -0.5676382184028625, -0.17053848505020142, -0.4540855884552002, -2.6002116203308105, 0.5709611177444458, -0.24078452587127686, -0.68528151512146, 2.5811147689819336, 0.10445863753557205, -0.6065776348114014, -0.31954237818717957, -0.6810837388038635, 0.6463186740875244, -1.1478270292282104, 3.1819894313812256, 0.6529814004898071, 1.0536649227142334, 0.8375600576400757, 1.6746008396148682, 0.10692796111106873, -1.144087314605713, -1.189146876335144, -1.7538145780563354, -0.20804272592067719, 2.162240743637085, 0.9147980809211731, -0.4370826482772827, -2.5846405029296875, -2.196216583251953, -0.54925537109375, 0.6605151891708374, -0.19399778544902802, -1.3934904336929321, 0.14226660132408142, 0.22585371136665344, 1.6717588901519775, 3.0736773014068604, -0.6617487668991089, 0.8801957368850708, -0.9145210385322571, 0.2758447825908661, -0.47378358244895935, -0.3503457009792328, 1.2503116130828857, -1.0730233192443848, -1.8652368783950806, -0.33554521203041077, -2.0697174072265625, -1.5116338729858398, 1.5527596473693848, -0.15479132533073425, -0.5430982112884521, -1.8702656030654907, -0.31545138359069824, -0.4360906779766083, -0.7633314728736877, -0.0019396285060793161, -0.31364938616752625, 0.3295121192932129, 0.23890705406665802, 1.252701997756958, -0.6886693835258484, -0.8014563918113708, -0.8398765921592712, -0.14838822185993195, -0.5509068965911865, 0.5987340211868286, 1.5632067918777466, 0.962192177772522, -1.1160697937011719, -0.12471991777420044, -0.6525467038154602, -0.8731191158294678, 0.1470944732427597, 0.5747941732406616, 0.8663025498390198, 0.21514058113098145, -0.8245996236801147, 0.0961613655090332, 0.44766032695770264, -0.9810804128646851, -0.8388566374778748, 0.26018962264060974, 0.5475080609321594, -1.464795708656311, -0.6595470905303955, -0.930952250957489, 0.772483229637146, 1.2146811485290527, 1.0916264057159424, 0.21432870626449585, -1.6719841957092285, 1.4852030277252197, -0.8099521994590759, -1.9352673292160034, -0.9333491325378418, 1.4028635025024414, 1.0789059400558472, 1.4680545330047607, -0.4357515275478363, -1.5880314111709595, 0.2842789888381958, 0.9866324663162231, 0.27368032932281494, -0.010130473412573338, 0.3701798915863037, -1.281138300895691, -0.5111348628997803, -0.35717687010765076, 0.14898933470249176, 1.262971043586731, -0.04417720064520836, 2.019275426864624], [-0.926596999168396, -0.422541081905365, 0.8650330305099487, -0.5716698169708252, -0.3640124499797821, 0.3447551727294922, -0.8527476787567139, 0.4148639440536499, -0.36612746119499207, -1.0744613409042358, -0.6921862959861755, 0.03432726114988327, -0.8112389445304871, -0.08419937640428543, 1.5286495685577393, 0.08043015003204346, 1.417167067527771, 0.4503672122955322, 0.8903393745422363, 0.8484010696411133, 0.869613766670227, 0.7636463642120361, -0.6169378757476807, -1.3404700756072998, -0.0033359061926603317, -1.3519365787506104, 0.6415392160415649, -0.1663099080324173, -0.3134535551071167, -0.38847383856773376, 0.28336912393569946, 0.9100525379180908, 0.9204249382019043, 0.15440179407596588, -0.22439852356910706, 0.1703348606824875, -0.9967156052589417, -0.6139271855354309, -0.022460319101810455, 0.8563187122344971, 0.2139279842376709, -0.156615749001503, -0.1956038773059845, -1.2899551391601562, -0.03502650558948517, 0.37606048583984375, -0.32806655764579773, 0.29534026980400085, -0.10021156072616577, 0.7674620747566223, -0.6544336676597595, 0.0301953237503767, -0.08927889168262482, -0.535365879535675, -2.158698558807373, 0.6260600090026855, -0.2135995328426361, -0.7042791843414307, -0.2450701892375946, 2.824496269226074, 0.11174163222312927, -0.7898096442222595, 1.632393717765808, -0.23402781784534454, -2.7927608489990234, -0.3521234691143036, -1.1502251625061035, 0.21206699311733246, 0.6033467650413513, 1.1958889961242676, -0.9612026214599609, -1.1775723695755005, 0.11649208515882492, 0.6931413412094116, -0.2048463225364685, 0.0004274844832252711, -1.0954551696777344, -0.8076611757278442, -0.2794082760810852, 0.936234176158905, -0.15840311348438263, 0.15810488164424896, 0.9376841187477112, -0.9014680981636047, 0.765900731086731, -0.2602880299091339, 2.0731849670410156, 0.6740325689315796, 2.0914628505706787, 0.8550918698310852, 0.16324187815189362, 0.6848181486129761, -0.633004903793335, 1.0426414012908936, 0.008296117186546326, 0.05242563411593437, 0.15293312072753906, 0.09853308647871017, -0.45394638180732727, 0.8100224733352661, -1.6291465759277344, 0.24075260758399963, 0.35624486207962036, 0.613297164440155, 0.8436447381973267, 0.09192803502082825, -0.98762047290802, -0.16879414021968842, -0.0422838069498539, -1.6389232873916626, 0.12571978569030762, -2.610013246536255, 1.6495585441589355, -0.1433030068874359, 0.8111469745635986, 0.24831590056419373, 1.3126721382141113, -1.3811582326889038, -0.3627096712589264, 0.4115290641784668, -0.8751124739646912, 1.5411802530288696, -1.5256049633026123, -0.5538002848625183, -1.3961604833602905, -1.4265198707580566, -1.09523606300354, -0.29033949971199036], [1.9975128173828125, -0.056826137006282806, 0.27947044372558594, 1.23307204246521, 0.6681020855903625, 2.0597379207611084, 2.980778455734253, -1.0066028833389282, -0.2046925574541092, -0.14229221642017365, 1.1832048892974854, -0.8798092007637024, -0.8140751123428345, 0.003171368967741728, -0.20675252377986908, 1.4072362184524536, 1.2444361448287964, -0.3212948143482208, -0.03012828156352043, 0.07113399356603622, -2.329411506652832, -1.7598137855529785, 0.7204994559288025, -2.4344263076782227, -1.2711890935897827, -1.0366970300674438, -0.041701458394527435, 0.28270477056503296, -0.4009542465209961, 0.6574774384498596, 0.6197068691253662, -1.394580364227295, -1.2193481922149658, -0.32195448875427246, -0.5322324633598328, -0.051465123891830444, -0.252730131149292, -0.5075339078903198, 0.1498185396194458, -0.34999197721481323, 0.8434576988220215, 0.5166369676589966, -1.7555996179580688, 0.2471274733543396, 1.485539197921753, 0.4473431706428528, -2.287020444869995, 1.337153434753418, -1.1046407222747803, 1.8301939964294434, -0.7376194596290588, -0.7891669869422913, 0.12553724646568298, 1.21176278591156, 0.2803889214992523, -0.6005274057388306, 0.34575632214546204, 0.797712504863739, -1.2729476690292358, -0.89711993932724, 0.2378406524658203, 0.2582409381866455, 1.8023868799209595, 0.13755592703819275, -0.24836014211177826, 0.11085651814937592, 0.191806823015213, 0.012818931601941586, -0.29942670464515686, 0.7152671813964844, -1.317432165145874, 0.3544274568557739, 1.5219776630401611, 0.6242486238479614, -0.19251269102096558, 0.39382433891296387, -0.9546547532081604, -0.24014118313789368, -0.19074194133281708, -1.5585224628448486, 0.671149730682373, -0.7834279537200928, 0.4097065031528473, -0.7066460847854614, 1.378553867340088, 0.13849085569381714, -0.11433272063732147, 0.2040991187095642, -0.6140810251235962, -0.7504740357398987, 1.341715931892395, -0.22954386472702026, -1.0071978569030762, 1.6039981842041016, -0.18397772312164307, -0.6877329349517822, -0.12059567123651505, 0.18145349621772766, 0.4371251165866852, -2.0247886180877686, 1.6971498727798462, -1.2937690019607544, -0.8197034597396851, 0.31713154911994934, 0.765323281288147, 0.5039919018745422, -0.04117438569664955, -0.44356033205986023, 0.5783794522285461, -0.29941290616989136, -0.8358032703399658, 1.3349767923355103, -0.3663065433502197, 0.3364136219024658, 0.5086880922317505, 0.8469926714897156, -0.40605902671813965, 0.9739205837249756, 0.2622363269329071, 1.512715458869934, 1.2222133874893188, 0.9723448753356934, 0.7572200894355774, -2.6597466468811035, -0.2935994565486908, -0.5297064781188965, 0.1805686056613922, 1.8223243951797485], [-1.9565192461013794, -0.02187226712703705, 0.5526830554008484, -0.16898910701274872, 0.15040116012096405, -1.1609586477279663, -0.6334825158119202, -1.0329978466033936, -0.6410517692565918, 0.7230572700500488, 0.0711149275302887, 0.5507219433784485, -0.09602192789316177, 1.3682398796081543, 0.5909014344215393, 0.051138147711753845, 0.33702021837234497, 1.1876217126846313, -0.5512771010398865, -0.14447003602981567, -0.46227094531059265, -0.2606242001056671, -0.2938423156738281, 0.8291589617729187, -1.304474949836731, 0.8939625024795532, -0.4724578857421875, -1.394156575202942, -1.1414527893066406, 0.09920133650302887, -0.004567279946058989, 1.3743137121200562, -1.156123161315918, -0.5809243321418762, -1.2066112756729126, -1.2711905241012573, -0.061181243509054184, 0.6517888903617859, -0.800593376159668, 1.2914241552352905, 0.3056289851665497, 0.18309380114078522, 2.8519513607025146, -0.6685591340065002, 0.9146896600723267, 0.03029804117977619, -0.37284478545188904, 1.1561704874038696, 1.199155569076538, 0.45201534032821655, 2.5796658992767334, 0.3378121256828308, 0.1687568724155426, 0.6088156700134277, 0.6052511930465698, 0.5717740058898926, -0.6034125089645386, -0.1176404282450676, 0.6712243556976318, 0.7692337036132812, 0.6162450909614563, -1.3271880149841309, 0.6738114356994629, 0.6889819502830505, -0.19229397177696228, 0.04249711334705353, 0.83053058385849, -1.6353965997695923, 2.159803867340088, 0.43075692653656006, -0.08955435454845428, -0.04712125286459923, 0.6707060933113098, -0.05147331580519676, -1.108959436416626, 0.39533764123916626, -0.3167250454425812, -0.4353407919406891, 0.9265892505645752, -0.384378045797348, -2.1957175731658936, 1.5333497524261475, -0.5935874581336975, 0.816700279712677, 1.1333013772964478, -0.09783732891082764, -0.5056440830230713, 0.028426596894860268, -0.5148828029632568, 0.9759223461151123, 0.8649300336837769, 0.1741025298833847, -1.5871468782424927, -0.5599095821380615, 1.4219300746917725, -0.7328760623931885, -1.2328165769577026, -1.6053986549377441, -0.6225056052207947, -1.2302830219268799, 0.2470618039369583, -0.55918949842453, -1.416837215423584, 0.8514336943626404, -0.691081166267395, 1.105254054069519, -0.29214510321617126, 0.25349557399749756, -0.5892441868782043, -1.3460818529129028, 0.29031017422676086, -0.12074894458055496, -0.6002455949783325, -0.7902063131332397, 0.2532210052013397, -1.6234302520751953, -0.5461024641990662, 0.10087791830301285, -0.8085764050483704, 1.1774498224258423, -2.31321120262146, 0.92789226770401, 1.0782586336135864, -0.3928154706954956, 0.7538931369781494, 0.055608637630939484, 0.9519078135490417, -0.908057689666748], [1.1069978475570679, 0.8264669179916382, -0.37049755454063416, -0.6951740980148315, 1.661455512046814, -1.2523548603057861, 1.0684531927108765, 0.46244707703590393, 0.16785690188407898, 0.4849928021430969, 0.7984657883644104, -1.1045492887496948, -1.6070425510406494, 0.24663065373897552, 0.1383877843618393, 0.834835410118103, -0.237540140748024, 1.4258962869644165, -0.5822167992591858, -2.4078550338745117, -0.2622601091861725, -1.925137996673584, -1.0458002090454102, 1.1700754165649414, -0.229569211602211, -0.8287023305892944, 0.6792439818382263, -0.3399292826652527, -1.341437578201294, -0.4163462817668915, -1.2084980010986328, -0.36358392238616943, 1.1793135404586792, -0.12522420287132263, -0.9243224859237671, -1.1377099752426147, -0.39660879969596863, 2.393662214279175, -0.7371578216552734, -0.8163797855377197, 0.26339519023895264, -1.0076714754104614, 0.04351144656538963, -1.8189337253570557, -1.1521379947662354, 1.9650086164474487, -0.3711545467376709, 1.2314555644989014, -0.09731657058000565, -0.5308448076248169, 0.05885232239961624, -1.2455105781555176, 0.12858766317367554, -0.05199853330850601, 1.2969080209732056, 0.234497532248497, 0.4525187015533447, 2.1943612098693848, -0.2430848330259323, -0.13468602299690247, -0.6693718433380127, -0.21963007748126984, 0.2527795135974884, 1.6988073587417603, -1.2049487829208374, -0.7115169763565063, -0.02240181155502796, 0.75433748960495, 0.9715186357498169, 0.8114479780197144, -0.8397619724273682, 0.6722663640975952, 0.01195472851395607, 0.04674932360649109, -0.32816585898399353, 0.3357747793197632, -1.625604271888733, -0.8602596521377563, -0.6436725854873657, -2.072108268737793, 0.6876793503761292, 0.6028656363487244, 1.199894666671753, 0.4862738847732544, 2.20621657371521, -1.340400218963623, 1.008809208869934, -1.863813877105713, 1.717093825340271, -0.22227874398231506, -0.7605892419815063, -0.16026070713996887, 0.16895230114459991, 0.702458918094635, 0.9044440388679504, -1.2808095216751099, -0.5554654002189636, -0.1887960433959961, -0.7910043597221375, 0.9783775210380554, -0.7031047940254211, -1.6854203939437866, 0.0342080257833004, 1.1431357860565186, -0.5081846714019775, -0.5089244246482849, -0.32325661182403564, -1.3633753061294556, -0.6217721104621887, 0.33647799491882324, 0.23946595191955566, 2.448838233947754, -0.06305268406867981, 0.20560579001903534, 0.027380075305700302, -0.8489142656326294, 0.7764206528663635, -0.4782031774520874, 0.7616060972213745, 0.10723871737718582, -1.5276594161987305, 0.3732438385486603, 0.9180463552474976, -1.3541343212127686, 1.4169493913650513, 0.8625967502593994, -0.4538177251815796, 1.0699957609176636], [0.7868616580963135, 0.45749107003211975, -1.2200502157211304, -1.1680498123168945, 0.08776307851076126, -0.5231244564056396, -1.125124454498291, -0.7947250008583069, 0.596686840057373, 0.8505003452301025, -0.4055578112602234, 1.4557039737701416, 0.5825892686843872, 0.011885746382176876, 0.8837677836418152, 1.558396816253662, 0.4953148663043976, -2.0059523582458496, -1.1074739694595337, 0.4225018620491028, -1.5962918996810913, -0.6367225646972656, 0.3698185086250305, -0.5501846075057983, 1.101681113243103, 0.5016711354255676, -0.30736660957336426, 0.9129266738891602, -1.4364405870437622, -0.6057685017585754, 0.4364194869995117, -0.8236686587333679, 0.07249945402145386, 0.8055598735809326, 0.5942201018333435, -1.3700073957443237, 0.1472860723733902, -0.2316327840089798, 0.2585599422454834, 0.006361289415508509, 0.09039923548698425, -0.7257371544837952, 0.5916891098022461, 0.8150728940963745, -0.4908939301967621, -1.5252795219421387, -1.494140386581421, -1.814149260520935, -0.366990864276886, -1.0997979640960693, 0.1824372261762619, -1.4345059394836426, -1.0386106967926025, -0.04102218523621559, -0.6568408608436584, -1.2811455726623535, 0.02664668671786785, -0.6952641010284424, -0.02098161168396473, -0.321800172328949, 1.5861544609069824, 1.7110772132873535, -1.2316259145736694, 1.7285234928131104, 0.5631247162818909, -0.04321535676717758, -1.3448235988616943, 0.36543068289756775, -2.6393768787384033, 2.2115561962127686, -0.7375256419181824, 0.6418821215629578, -0.6171380281448364, 0.3145092725753784, -0.14898037910461426, 0.1825902909040451, 1.3219996690750122, 0.4139239192008972, -1.0920720100402832, -1.496592402458191, -1.4453684091567993, -0.11477493494749069, 0.8163623213768005, 0.6894641518592834, 0.7903963327407837, 0.9518983364105225, 0.9263201355934143, 0.25529786944389343, 0.5204135775566101, 0.13766834139823914, 0.42808645963668823, -0.5421414971351624, -0.01281761284917593, 0.5241172313690186, -0.06068629398941994, 1.0436055660247803, 0.7341929078102112, -0.9693706035614014, -0.8456896543502808, 0.6607226729393005, -0.3885364830493927, 0.0025427655782550573, -0.9773417711257935, 0.1382666528224945, -1.015517234802246, -1.352328896522522, -0.28823286294937134, 1.5347168445587158, 0.9053560495376587, 1.947392225265503, 0.39391565322875977, 0.1448812633752823, -0.29962456226348877, 2.1358630657196045, 1.4996626377105713, 0.46085798740386963, -0.4716319739818573, 1.597914695739746, -1.2240517139434814, 0.36069855093955994, -2.4318532943725586, -0.19004769623279572, 0.7792468667030334, -1.1526607275009155, 1.2703585624694824, 2.1232988834381104, 0.7149300575256348, -0.5228609442710876], [0.15947295725345612, 1.2454826831817627, -0.9876230359077454, -1.174161672592163, 0.7727431058883667, 0.18061880767345428, 0.8114404678344727, -0.8448620438575745, 1.6621726751327515, -0.030809221789240837, -0.9686986207962036, -1.0224450826644897, 1.2909948825836182, -0.08389855921268463, 1.137896180152893, -1.9221131801605225, -0.9627647399902344, 0.9509420394897461, -0.39939913153648376, -0.9345201253890991, -0.6379724144935608, 1.038740634918213, -0.08799134194850922, -0.39771515130996704, -0.3113649785518646, 0.07791934907436371, 1.2929003238677979, -0.743751049041748, -0.9290850758552551, -0.5920833945274353, -1.486378788948059, -0.1137256771326065, 0.4758647382259369, -0.17134620249271393, -0.24816708266735077, 2.1710081100463867, -0.8763564229011536, -2.643846035003662, 0.15317532420158386, 0.09876884520053864, 0.98875892162323, -0.9229230880737305, 0.7829442620277405, -0.3190309405326843, -1.2505074739456177, 0.9480518698692322, -0.16035433113574982, -0.8059355616569519, -1.0596753358840942, -1.2294070720672607, 0.16503660380840302, 0.2429582178592682, 0.05963483452796936, -1.8295649290084839, 1.3189548254013062, -1.3730813264846802, 0.570341169834137, 0.8258615732192993, -2.770380735397339, 0.9596796631813049, 0.9387661218643188, 0.7304481863975525, -0.18855643272399902, 0.27562156319618225, 0.7953096032142639, -0.05103100463747978, 1.0652005672454834, -1.4884949922561646, 0.6798794865608215, -2.348921775817871, 0.8388240337371826, 0.9076080918312073, 0.3072749078273773, 0.002166177611798048, 1.1400549411773682, 0.04183585196733475, 0.7344988584518433, -0.11367201060056686, 0.9163715839385986, -0.36440953612327576, -1.5470046997070312, 1.6013824939727783, -0.8549374341964722, 1.5086514949798584, 0.009722677990794182, 0.8299731612205505, 0.13107480108737946, -0.11281906813383102, 1.319860577583313, 0.013978996314108372, 1.2001157999038696, -0.4315066337585449, -0.7778300642967224, 0.03566187620162964, 0.1975427269935608, 0.5323827862739563, -1.8313488960266113, -0.051373645663261414, 0.40173110365867615, 1.7160090208053589, 0.21719413995742798, 0.42273202538490295, -0.22591359913349152, 2.952516794204712, 0.02251669019460678, 0.2728956937789917, 0.048270583152770996, -1.651281714439392, 0.20531336963176727, -1.3803274631500244, 0.6075617671012878, -2.8234570026397705, 2.037956476211548, -1.00980544090271, -0.3584810197353363, -0.4569615125656128, 0.5312120318412781, 0.9861026406288147, -0.5211532711982727, 1.2177183628082275, 0.10748474299907684, 1.280913233757019, 0.8346617221832275, 0.556205153465271, -0.8578832149505615, 1.4400736093521118, -3.066606283187866, -1.046533226966858], [-0.6258008480072021, -0.23625649511814117, 1.2014695405960083, 0.1528787463903427, -1.6128236055374146, -0.4833351671695709, 1.4277323484420776, 0.8340650796890259, -0.5778266787528992, -0.09559784084558487, 1.2294838428497314, -0.39308440685272217, 0.11564837396144867, -0.4412427544593811, -1.5661494731903076, 1.1542960405349731, -0.8287347555160522, 0.11497589200735092, 0.6509788036346436, -1.0604918003082275, 0.2248496264219284, -0.8897160291671753, 1.3566738367080688, -1.391953706741333, -1.287743330001831, -0.5231711268424988, -0.6818554997444153, -0.405266672372818, 0.4977400302886963, -0.5351660251617432, 0.7798973321914673, -1.4413864612579346, 1.3941614627838135, 0.2788461744785309, -0.658332347869873, -0.9317545890808105, 0.4122232496738434, 0.36907100677490234, 1.9128494262695312, 0.10647273063659668, 1.0389457941055298, 0.009663259610533714, -0.3077285587787628, -1.122637152671814, -0.8606592416763306, 0.02414468489587307, 0.40859225392341614, 0.6892943382263184, 0.2521013021469116, -0.6264330744743347, -0.5007441639900208, 0.7978570461273193, 0.555703341960907, -1.280672311782837, 0.48069527745246887, -0.4264509081840515, 0.08883308619260788, -0.4376184046268463, 1.243764877319336, 0.4600996673107147, 0.512706458568573, 0.2297709733247757, -0.099027119576931, -0.23260119557380676, -0.9718952775001526, 0.24422287940979004, -0.9409699440002441, 0.5453519821166992, -0.03035752847790718, 0.212980717420578, -0.4529520869255066, 1.002704381942749, -0.23147182166576385, 0.6052400469779968, -0.9021997451782227, 1.7871886491775513, 0.2716990113258362, 0.5287309885025024, -0.6016189455986023, 0.3410912752151489, 0.03630445525050163, 0.6800901293754578, 0.7872609496116638, 1.5601460933685303, 2.0953409671783447, -0.3060168921947479, 0.1652214676141739, -0.6902326345443726, -0.6660192012786865, -2.002525568008423, -1.4237645864486694, -0.3757295608520508, -1.7049627304077148, 0.8197900652885437, 0.8697239756584167, -0.421806275844574, -0.2616705000400543, 1.0249838829040527, 1.66177499294281, 0.2462027668952942, -0.07907728850841522, -0.18115639686584473, -1.785416603088379, 0.4156492054462433, -0.6685956120491028, -0.48661503195762634, -0.4697469174861908, -2.2258198261260986, -1.1498934030532837, -0.05233515053987503, 0.13782908022403717, -0.12315675616264343, 1.0665467977523804, 1.2490969896316528, 1.4254359006881714, 0.9317101240158081, 0.7368716597557068, 0.6187567114830017, 0.9387617111206055, -0.4960884153842926, -0.05898779258131981, 0.7630853652954102, 0.4910706579685211, 0.42627257108688354, 0.892035961151123, -2.1289405822753906, 0.40173056721687317, -0.4702025055885315], [0.24034833908081055, -0.5921466946601868, -1.5328410863876343, -0.7853347659111023, -0.2925299406051636, -0.19375094771385193, 0.5377582907676697, 1.1965938806533813, -0.9836156964302063, 0.26212817430496216, -0.4618532061576843, 0.5325379371643066, 0.43883374333381653, 0.59602290391922, -2.222550392150879, 0.27653101086616516, 1.573559284210205, -1.374273657798767, 0.39908793568611145, -0.3008616864681244, -0.42189574241638184, 0.6227395534515381, 1.3593930006027222, 0.7417345643043518, 1.5241978168487549, -0.5965118408203125, 1.3668978214263916, -1.0423169136047363, -0.23177459836006165, 0.8709742426872253, -1.5480947494506836, 0.1097477376461029, 1.2862114906311035, -1.1639149188995361, 0.42868945002555847, -0.09979035705327988, -1.2503719329833984, 2.100687026977539, 0.020138509571552277, 0.3151625394821167, -0.29582130908966064, 0.6853501796722412, 0.7266222834587097, -0.49633651971817017, 0.5754407048225403, -2.149583339691162, -0.9787327647209167, 0.23049092292785645, -0.8634735941886902, -1.5828118324279785, -0.19982559978961945, -0.5805479288101196, 0.48762989044189453, -0.6292324662208557, 1.0705780982971191, -0.6533297300338745, -0.6631169319152832, 1.0497241020202637, -0.12200944125652313, 0.18530842661857605, 0.7847468852996826, 1.3907934427261353, 1.6991512775421143, 1.6467578411102295, 1.4682530164718628, 0.786673903465271, 1.1400632858276367, -0.37729695439338684, 0.6356887221336365, -0.6270962357521057, -0.11336303502321243, -1.0656609535217285, -0.47863009572029114, 0.7045500874519348, 1.3966819047927856, 1.1639422178268433, -1.5243611335754395, 1.007758378982544, -0.8479716777801514, 1.9373334646224976, 0.8604797124862671, -1.8385123014450073, 0.8969115614891052, 0.3125726878643036, -0.2782122790813446, 0.17952273786067963, -0.12345249950885773, -0.06457575410604477, -0.011594394221901894, -0.8553733825683594, -0.6934311985969543, -0.018041687086224556, -0.9900569319725037, 0.36590686440467834, 0.08208017796278, 0.7456742525100708, 0.6070771217346191, -0.36787331104278564, 1.0487251281738281, -0.6529297828674316, -0.43928462266921997, -0.4746100604534149, -0.04494849219918251, 0.008549570105969906, -0.4031069874763489, 0.15899312496185303, -0.9048752784729004, 0.8197560906410217, -0.5077735185623169, -1.5105398893356323, 0.4640621840953827, -1.7604684829711914, 0.29988113045692444, -1.141051173210144, -0.019194576889276505, 0.9295248985290527, 1.0441814661026, -0.7964407205581665, 0.36443284153938293, 0.6745293140411377, -0.20365476608276367, -1.3253731727600098, -1.3864742517471313, 0.3497776985168457, 0.20735985040664673, 0.8721945285797119, -0.47792181372642517, -0.17377561330795288], [0.06034566089510918, -1.3062790632247925, -0.727202296257019, -0.6286082863807678, 0.05900321528315544, -0.10602696985006332, 0.18886932730674744, -1.3105956315994263, -0.03119461052119732, -0.767756462097168, -0.25061604380607605, 0.7888616323471069, 0.11557155847549438, 1.7716084718704224, -0.2174633890390396, -1.3455051183700562, 1.3307406902313232, 0.8276476263999939, 2.0331871509552, 0.950128972530365, 2.4858744144439697, -0.3094899654388428, -0.3069223165512085, -0.7535107135772705, 0.07473520934581757, 0.7467119097709656, 0.7018478512763977, 1.4897983074188232, -0.3342802822589874, 1.5536587238311768, 0.9526747465133667, -0.8368034362792969, -2.2447047233581543, -1.6810916662216187, 0.12129025161266327, -1.2116153240203857, -0.42182475328445435, 0.6750769019126892, -0.5484822988510132, 0.2890946865081787, -0.05480044335126877, 0.5622469782829285, 1.1936235427856445, 0.888476550579071, -0.6345211863517761, -0.9198790192604065, 0.1124320924282074, -1.0812571048736572, -0.1896621733903885, 0.8055906295776367, -0.2270595282316208, -0.8926184773445129, 1.1032204627990723, -1.3108234405517578, -0.15297307074069977, -0.7206860780715942, -0.212494358420372, -0.08742699772119522, 0.3681841790676117, 0.26450595259666443, -0.3151041269302368, -0.2393004149198532, 0.418661504983902, -0.3896137773990631, -1.363318920135498, -0.16608110070228577, -1.2200937271118164, -0.700789749622345, 0.5512447953224182, 1.5332262516021729, 0.9689926505088806, -1.257097601890564, -0.0347101055085659, -0.38060232996940613, -0.9403342008590698, -1.173184871673584, -0.8810267448425293, -0.6822088956832886, 2.7281999588012695, 0.2197076827287674, 1.9446089267730713, -0.8378973603248596, -1.4269729852676392, -0.9784367084503174, -0.026791604235768318, -0.17805713415145874, 1.241563081741333, 1.4043762683868408, -1.2541707754135132, -2.1647846698760986, 0.9364403486251831, 2.0180718898773193, -0.5987328290939331, -2.394218921661377, -0.5041545033454895, -1.268480896949768, 0.4946979582309723, -1.149071455001831, -1.2516005039215088, 0.1681351661682129, -0.4055248200893402, 0.5364227294921875, -1.450777530670166, -2.1959750652313232, 0.24212661385536194, -0.12139751762151718, -1.1529626846313477, 1.0977689027786255, -0.2140946090221405, 0.29864513874053955, 2.1887574195861816, 1.2049672603607178, 0.8377155065536499, -0.015338817611336708, -0.6705856919288635, 0.771377444267273, -1.4752779006958008, -1.285398006439209, -0.21813002228736877, -0.6292340755462646, -1.6297794580459595, 0.5799867510795593, -0.19412511587142944, -0.8105382323265076, 1.4738372564315796, -0.5542325973510742, 0.13571622967720032, -1.3789188861846924], [-1.8909105062484741, 0.7637394666671753, 2.2420547008514404, -0.6251180171966553, 0.6588319540023804, -0.27806299924850464, -0.3021336793899536, 0.2812468409538269, 2.469237804412842, 0.23772719502449036, -0.5511268973350525, 0.0018144453642889857, -0.5578483939170837, 1.090658187866211, -0.671823263168335, -0.5600232481956482, 1.435182809829712, -1.402191162109375, -1.404395580291748, -3.232621431350708, 1.4420312643051147, -0.01926976442337036, -0.7567192912101746, -1.5597885847091675, -1.7521121501922607, -0.5386849045753479, 0.5072098970413208, -1.080540418624878, 0.8285738229751587, 1.634223222732544, 0.8810492157936096, 0.7353432774543762, -0.20962105691432953, -0.5931849479675293, -0.5703744888305664, 0.49629828333854675, 1.4295109510421753, -0.5925074219703674, -0.9006754755973816, -0.31588679552078247, -0.6983896493911743, 0.33156535029411316, -1.0157747268676758, -0.49633321166038513, 1.9490406513214111, -0.4548678994178772, 0.8502278923988342, 0.4844950735569, -1.0681933164596558, 0.23255346715450287, -1.234901785850525, 1.6031984090805054, -0.7550888657569885, 0.04174179583787918, 0.8952399492263794, -1.4088255167007446, 0.9573739767074585, 0.720250129699707, -1.7626984119415283, 0.9284008741378784, -0.8565475344657898, -1.3182406425476074, 1.116366982460022, -0.7419258952140808, 0.43281298875808716, -0.7316697835922241, -1.4774335622787476, -0.011686409823596478, 1.149385929107666, 0.4385707378387451, 0.7742685675621033, 0.5695416331291199, -1.0316671133041382, 0.45252725481987, -0.5448449850082397, 0.9223396182060242, 0.2888718545436859, -0.01990761235356331, -1.5897800922393799, 0.6558716893196106, 0.13984107971191406, 0.3776554763317108, -0.09266462922096252, -1.607864260673523, 1.6700294017791748, 1.3729392290115356, 0.4338246285915375, 0.2978156805038452, -1.4824377298355103, -0.8279744982719421, 1.8178091049194336, -0.6251817941665649, -0.6662032604217529, 1.5281269550323486, 0.6688340306282043, 0.39148151874542236, -0.3747066557407379, -1.657883882522583, -0.7768533229827881, 1.5049216747283936, -1.024675965309143, -1.0706920623779297, 0.03537419065833092, 0.24088633060455322, -1.0406006574630737, 0.6572099924087524, -0.6767301559448242, 1.1097081899642944, -0.3367990255355835, 0.7007246613502502, -0.45173966884613037, 1.1560837030410767, 1.912641167640686, 0.03710845485329628, 0.20489385724067688, -0.19862128794193268, -0.9140001535415649, 0.8785304427146912, -0.6877099871635437, 0.7117132544517517, 0.44381746649742126, 0.6567617654800415, 1.0017188787460327, -1.1566390991210938, -1.6654537916183472, -0.8799634575843811, 0.21948005259037018, -0.5568313002586365], [0.058742955327034, -0.4038262367248535, -1.0667108297348022, -1.4504667520523071, -0.2468983680009842, 1.1879850625991821, 0.6416922211647034, -0.937882125377655, 0.8575531840324402, 0.13458681106567383, 1.3405128717422485, -1.0296694040298462, -0.8325653076171875, 0.15641257166862488, 1.4076998233795166, 0.07911165803670883, -1.6943697929382324, 0.22322140634059906, 0.13599303364753723, 1.2014156579971313, -2.457549810409546, -0.14830993115901947, -0.33023712038993835, -0.6775620579719543, 0.518757164478302, 1.6554285287857056, -0.05624011158943176, -0.19774043560028076, -0.23232963681221008, -1.0538413524627686, -1.3869640827178955, -0.1551792025566101, -0.6623595356941223, -0.03995984420180321, 0.1049443930387497, -2.483935594558716, -0.05690791457891464, 0.2989574372768402, -0.5116718411445618, -0.5901382565498352, -0.7631017565727234, 0.6613089442253113, 1.6612917184829712, 1.1826717853546143, -1.1402047872543335, 1.443457841873169, 0.3651772141456604, 0.8265838623046875, -0.331373393535614, 0.5272417068481445, -0.372480571269989, -0.627297043800354, -1.3676817417144775, -0.28518742322921753, -0.36141547560691833, 2.0146703720092773, -0.32485294342041016, -0.14393861591815948, 0.27033036947250366, -1.552068829536438, 0.5669229626655579, -1.9690349102020264, -0.7021369338035583, -0.6851223111152649, 0.1841958463191986, 0.42077505588531494, 0.8115569353103638, -1.4758665561676025, 1.9659826755523682, -0.8685801029205322, 0.4996902346611023, -0.26704058051109314, 0.46543124318122864, -1.2950305938720703, -0.38661491870880127, 0.7368629574775696, 2.0005836486816406, 0.4770875573158264, 0.838066041469574, -0.5924615263938904, 0.12142474204301834, -1.5816506147384644, -1.4755988121032715, 0.9873659014701843, 0.8642191290855408, -1.093122959136963, -1.3332173824310303, -0.358757883310318, -1.2441977262496948, 1.7846800088882446, -0.40415993332862854, -0.9403076171875, -1.4552606344223022, 0.40946799516677856, 0.33528637886047363, 0.3629525899887085, -0.263871967792511, -0.26769423484802246, -1.3016353845596313, 0.08633892238140106, 0.4925484359264374, 0.059752609580755234, -1.0236014127731323, 0.346367746591568, -0.44399482011795044, 1.7481062412261963, -0.7348645329475403, -0.30702710151672363, -0.2627873718738556, 0.9084323644638062, -0.019290171563625336, -1.253319263458252, 0.06021939963102341, 0.07500359416007996, 0.0049058725126087666, -0.5490778088569641, -1.8673650026321411, -1.7878577709197998, 1.0878393650054932, -0.7766659259796143, -0.24879276752471924, -0.27411726117134094, 0.7989521026611328, -1.7271957397460938, -0.5689224600791931, -0.6331154108047485, -0.33027026057243347, -1.1473232507705688], [0.9277539849281311, 1.3373302221298218, 0.5755983591079712, -0.3847840130329132, -0.2610635757446289, -0.019707513973116875, -0.4253825545310974, -0.3127245306968689, 0.01911928690969944, 0.5564404726028442, 2.1705007553100586, -0.01627831533551216, -0.7415497899055481, -1.1711183786392212, 0.7508295774459839, 1.521704912185669, 1.358393907546997, -1.623783826828003, -0.5902991890907288, 0.22588300704956055, 1.157780408859253, -0.4543797969818115, -1.11432945728302, -0.6341018080711365, -0.11813497543334961, 0.7861631512641907, 0.7934750318527222, -0.36910560727119446, -0.4332933723926544, 0.5493459701538086, -0.18061333894729614, 0.897750973701477, -1.8356328010559082, 0.052434712648391724, 0.01880825310945511, 0.23549266159534454, 0.43505436182022095, 0.4108520448207855, -0.62404465675354, 1.281685471534729, 0.13348713517189026, -0.17327065765857697, -1.2616938352584839, 0.45343080163002014, -0.91579669713974, 0.4698035418987274, 1.227349877357483, -0.16929754614830017, 2.6947991847991943, -0.5377587080001831, 1.1181813478469849, -0.9940308928489685, -1.102534532546997, 0.7961552143096924, 0.9934210181236267, 0.5571804642677307, -0.7072086334228516, -0.9377694129943848, -0.967110812664032, -0.13574156165122986, 0.09134559333324432, -0.4833854138851166, -0.5909325480461121, 1.1685901880264282, -0.5711098909378052, -1.6061605215072632, -0.46634653210639954, -0.9391673803329468, 1.1744544506072998, 2.8073065280914307, -1.3771523237228394, -0.12557263672351837, -0.03167955204844475, -0.8784729242324829, -0.02359846793115139, 0.6769130229949951, -0.017462648451328278, -0.33907392621040344, 0.09446488320827484, 0.6554194092750549, -0.5303104519844055, -0.4453590214252472, 0.5869150757789612, -0.2231680452823639, -0.48398861289024353, -0.7217719554901123, 0.13365308940410614, 0.5335723757743835, 1.671256422996521, 0.09163729101419449, -0.5011847019195557, 0.17564274370670319, 1.1066409349441528, -0.7137181758880615, 0.21125657856464386, -0.8968459367752075, -0.1372002810239792, -1.0089937448501587, 0.8126571774482727, -0.154526486992836, -0.9171627759933472, -0.826965868473053, 0.3534483313560486, 1.374806523323059, 0.2512015998363495, -1.431106686592102, 0.8771611452102661, 0.538564920425415, -0.24244628846645355, -0.9265092611312866, -0.9779388308525085, -0.11994343996047974, 0.5479493141174316, 0.014184889383614063, 0.43360474705696106, 0.311468243598938, 2.126701831817627, 0.8688718676567078, 1.3187364339828491, 0.5039469599723816, -0.22488929331302643, -0.366731733083725, 0.15890009701251984, 0.12520650029182434, -0.3342887759208679, -2.0908076763153076, -0.44595035910606384, -0.01925724931061268], [-1.0499770641326904, 1.6963063478469849, 0.24715310335159302, -0.9224881529808044, 1.6368169784545898, 0.4176628887653351, 0.1417781114578247, 0.5614618062973022, 0.3986387848854065, 0.3924444615840912, 1.1798170804977417, -0.0609096921980381, 1.2814337015151978, 1.1367355585098267, -1.211159110069275, -1.4937667846679688, -0.43998992443084717, 2.1880900859832764, 1.0143015384674072, -0.711539089679718, 0.33332470059394836, 0.28677675127983093, 0.583763837814331, -0.22375917434692383, 0.8214386105537415, 1.248734474182129, 1.4887808561325073, 0.30074289441108704, 0.11891590058803558, 2.4067091941833496, 1.0541470050811768, 0.21252359449863434, 0.5155863761901855, 0.47661933302879333, -1.4936254024505615, 2.1472716331481934, 0.4291832745075226, 0.17026826739311218, 0.7184536457061768, -1.750259280204773, 0.32671788334846497, 0.058488063514232635, -0.32937905192375183, -0.9369823932647705, -2.4020633697509766, -0.3962878882884979, -1.0757856369018555, -0.38160043954849243, -0.7921841740608215, -0.21691547334194183, -0.6216343641281128, 0.2593580484390259, -0.03979204595088959, 0.6147798299789429, 0.4119711220264435, 1.566434383392334, -0.3327874541282654, -0.6044301986694336, 0.6612246632575989, -0.7761094570159912, 1.6015750169754028, -1.9979803562164307, 0.2166014462709427, -2.147840738296509, -0.28469547629356384, 0.9306268095970154, 0.6950960159301758, 0.8090885877609253, -0.4224577844142914, -0.5226364731788635, 1.6306005716323853, 0.1486804187297821, -0.006226575002074242, 0.7010831832885742, 0.008939986117184162, 1.0418415069580078, -0.11728273332118988, 0.698196530342102, -0.2600349485874176, -0.9658511281013489, 0.7400807738304138, 0.3059183657169342, 0.8573837876319885, -0.48589545488357544, -0.5849697589874268, 0.25877097249031067, -0.7306346297264099, -0.05925925821065903, 0.16116952896118164, -0.6337944269180298, 0.6892691850662231, -1.1009083986282349, -1.6133676767349243, -0.17872078716754913, 0.8606353998184204, 1.1641590595245361, 0.01852906309068203, 1.0062305927276611, 0.6344863772392273, -1.76161789894104, -0.602837085723877, -0.21655593812465668, 0.584708034992218, -0.01880417764186859, -0.6099315285682678, -0.05320535600185394, -2.3045153617858887, 0.12215150147676468, 1.386725664138794, 0.7080673575401306, 1.5279462337493896, -0.6659454703330994, -0.1862097829580307, 1.536380410194397, -0.5438223481178284, 0.7502274513244629, -0.2998594045639038, 0.04225912690162659, 0.2847990393638611, 1.4229798316955566, -0.4294343590736389, -0.7995113134384155, 0.039898671209812164, -0.03388811647891998, -0.5893085598945618, 0.7990780472755432, -0.2761279046535492, -1.166264295578003], [-0.3806115984916687, -0.2580738067626953, 0.7280280590057373, -1.0822527408599854, -1.2368414402008057, 2.17443585395813, -0.21548451483249664, -0.13315311074256897, 0.6780188679695129, 1.4004579782485962, 0.6110265851020813, -0.42755040526390076, -0.5014044046401978, -1.154394507408142, -0.6847654581069946, -0.9981098771095276, -0.5164142847061157, -0.4073157012462616, -0.047148074954748154, -0.36684080958366394, 1.3922532796859741, -0.5627878308296204, 0.4331890940666199, 0.4794539213180542, 0.2305862307548523, 0.7924861311912537, -0.3393392860889435, -0.8371546864509583, 0.23895764350891113, 0.3554460406303406, 1.005152940750122, 1.3193105459213257, 0.2858189046382904, 1.9778258800506592, 0.9762898683547974, -1.4760751724243164, -0.06804508715867996, -0.5149965286254883, -0.006155199371278286, -0.5272099375724792, -0.6238279342651367, 0.22612734138965607, 1.1099820137023926, 0.38383620977401733, 0.5005825161933899, -1.083385944366455, -1.005989909172058, -0.24207767844200134, -0.015167768113315105, -0.38818100094795227, -0.992631733417511, -1.938366174697876, -0.3869195580482483, -0.6126707196235657, -0.6698983311653137, -0.829291045665741, 0.5820607542991638, 0.37901875376701355, 0.4799312651157379, -1.1123526096343994, 0.3955884873867035, -0.8853632807731628, 0.5631686449050903, -0.7041202783584595, 0.16591084003448486, -0.5790864825248718, 0.164534330368042, -2.3276114463806152, 0.21267621219158173, 0.17056460678577423, 0.26230132579803467, 0.016691109165549278, -1.2830694913864136, 0.5107619166374207, 0.7500607967376709, 0.8150858879089355, 2.0399327278137207, -0.2105753868818283, 0.015922153368592262, 1.3369884490966797, -1.2537158727645874, -0.3518948256969452, -1.189415454864502, 0.14891178905963898, -0.47345152497291565, 0.6333870887756348, -1.1831321716308594, -1.4168541431427002, -0.11777616292238235, -0.7440864443778992, -1.2632575035095215, 0.7722081542015076, -0.6330445408821106, 0.7148763537406921, -1.9543002843856812, -2.142690896987915, 0.6459258794784546, 1.3508999347686768, 1.477076530456543, -0.3516960144042969, 1.2444190979003906, 1.6866803169250488, 0.14506842195987701, 0.8492352366447449, -0.5045763850212097, -0.9501594305038452, 0.09507464617490768, 0.773736834526062, 0.5999709963798523, 0.6945662498474121, -0.6272123456001282, 0.4563405513763428, -0.5663946866989136, 0.09548069536685944, 0.16228361427783966, -0.7368833422660828, 1.030777096748352, -0.18743561208248138, 0.7371134757995605, 0.04637886583805084, 1.0207302570343018, -1.064548373222351, -0.883299708366394, -0.7848289608955383, 0.2137833684682846, 0.40963372588157654, 0.1933724284172058, -0.5510937571525574], [0.4793083667755127, 0.1813347488641739, -1.0254274606704712, 0.9788328409194946, -0.15472015738487244, 0.39959225058555603, -0.17551536858081818, -0.07024946063756943, 0.5026997923851013, 0.09866338223218918, -1.5425841808319092, 0.33607593178749084, 0.6546251177787781, 0.8743084669113159, -1.063414216041565, -0.2330031394958496, -1.3618043661117554, 0.8354871273040771, -0.10221260786056519, 0.22990216314792633, 0.13335967063903809, -0.6751506924629211, 0.5038847923278809, -1.6720197200775146, 1.026198148727417, 0.07792795449495316, -0.9016307592391968, 0.3419819474220276, -0.8093050718307495, 1.0881803035736084, -0.6666717529296875, -1.058628797531128, 0.9368295073509216, -0.7447307109832764, -0.4739742875099182, -0.7999778389930725, -0.40151435136795044, -1.488214135169983, -0.5309045314788818, 1.7374231815338135, -0.5808452367782593, 2.190237045288086, 1.5230330228805542, 1.6293249130249023, 1.236429214477539, 1.9989864826202393, -0.2501783072948456, -1.3347963094711304, 1.2909815311431885, -1.0727828741073608, -1.2736506462097168, 1.6076338291168213, -0.04180927202105522, -0.10131599009037018, 0.777655303478241, 0.7036446928977966, 0.4382518231868744, 0.9317468404769897, -0.03730305656790733, 0.7717049717903137, -0.14952366054058075, -0.45208147168159485, -0.8616507053375244, -0.3846520483493805, 0.7484858632087708, 0.9603344798088074, -1.1362909078598022, -0.8865826725959778, 2.0460124015808105, -1.38486647605896, -0.03759128972887993, -0.6813288927078247, 0.21676243841648102, 0.06461700052022934, -2.5902600288391113, 0.36098170280456543, -3.2438533306121826, -0.0330827496945858, 0.26431500911712646, 0.24914802610874176, -0.28352031111717224, 0.5883920192718506, -1.4311201572418213, -1.5162791013717651, 0.4601078927516937, -0.38496458530426025, -0.7239009737968445, 1.372241735458374, 0.4380878508090973, -0.6789109110832214, 0.4855101406574249, 0.5060569047927856, -0.09801024198532104, 1.2151012420654297, -0.28658002614974976, -1.0619922876358032, 0.5191508531570435, 0.8678157925605774, -0.8607243895530701, 0.7606489658355713, 0.9516905546188354, -1.040278673171997, -0.09834836423397064, -0.06700088828802109, 1.4390952587127686, -0.5981593132019043, 0.32496413588523865, 0.11865347623825073, -0.7594075202941895, -0.3180473744869232, -0.36862727999687195, 0.7099533677101135, -0.6558037996292114, -0.464385986328125, -1.1291396617889404, -0.5587353706359863, -0.5972458124160767, 0.10346557199954987, 0.724654495716095, 1.7280066013336182, 0.126506045460701, 1.0644469261169434, 1.0648057460784912, 0.10344705730676651, -0.6030572056770325, 1.6517436504364014, 1.5994176864624023, -0.019626393914222717], [-1.284563422203064, -0.3433539867401123, -0.9006718397140503, 0.2334093302488327, -1.1540923118591309, -1.9253367185592651, -1.33309805393219, 0.7741640210151672, 0.2685191035270691, -2.4715144634246826, 0.7123100757598877, -0.024586278945207596, 0.04104315862059593, 0.4485657215118408, -0.1410658061504364, -2.7037765979766846, -0.22415074706077576, 0.11322830617427826, 0.2699629068374634, -0.140615776181221, 1.337026596069336, 0.20820185542106628, -0.9405038356781006, 0.7217261791229248, -2.0937700271606445, 1.030724287033081, -1.3019347190856934, -0.8979656100273132, 0.2994645833969116, 1.1338621377944946, -0.6136900186538696, -1.668349027633667, 0.7653283476829529, 1.6121420860290527, -1.2596644163131714, -0.23200726509094238, 0.3291756212711334, 0.20218956470489502, -0.7553549408912659, 0.03761417046189308, -0.49358728528022766, -1.8544132709503174, 1.0467621088027954, -0.6144741773605347, -1.336863398551941, -0.13268832862377167, -0.2533799409866333, 0.6646499037742615, -0.39983275532722473, 0.09960505366325378, -0.24651625752449036, -1.2496507167816162, 0.4563809633255005, -2.476741313934326, 0.49007460474967957, 0.2839103043079376, 2.7656280994415283, -0.416856974363327, -1.4452025890350342, -0.05695941671729088, -0.33483049273490906, -0.8006356358528137, 0.6317355036735535, 0.5605005025863647, 0.14918695390224457, -1.5303171873092651, 1.6265513896942139, -1.9220058917999268, 1.7662968635559082, -1.5301867723464966, 1.76669442653656, -1.1058154106140137, 0.11412874609231949, -1.487765908241272, -0.8135301470756531, -0.8734153509140015, -1.3598415851593018, 0.13602620363235474, -0.9929325580596924, 0.0502297542989254, -0.3292441666126251, 0.35870426893234253, 2.0220553874969482, 0.3340425491333008, 0.5265278220176697, -1.4532254934310913, -0.749626100063324, 0.779941976070404, 0.15392017364501953, -0.6010417342185974, -0.17732885479927063, -0.339860737323761, 0.10595113784074783, -0.45342934131622314, 0.8654180765151978, -0.8994373083114624, -1.6407959461212158, -0.6348158717155457, 0.2008083462715149, 1.5489552021026611, -0.49181050062179565, 0.9037268161773682, -0.0777563750743866, -0.8173051476478577, 0.011837909929454327, 0.6910481452941895, 1.3194208145141602, 0.6106901168823242, 1.3707597255706787, -0.20024409890174866, -0.3889226019382477, -1.5041722059249878, 0.7041988968849182, 0.8445293307304382, 0.9288464188575745, -1.6451274156570435, 0.45093923807144165, -0.8462088704109192, 1.4634579420089722, -0.12596242129802704, 1.3833168745040894, -0.0287947878241539, 0.12843681871891022, 1.5854860544204712, -0.5844711661338806, -1.4377917051315308, -0.43659210205078125, -0.20028497278690338], [-2.2717268466949463, -0.4343350827693939, -0.2261786311864853, 1.290238857269287, 1.6794651746749878, 0.6603900790214539, -1.0141332149505615, 0.6193228960037231, -0.7400439381599426, -0.33201584219932556, -0.8464553356170654, -0.9220383763313293, -0.7266064882278442, -0.9794921875, 0.03141719847917557, 0.7162400484085083, -0.23886531591415405, -0.41718757152557373, -1.4348516464233398, -0.2090531587600708, -0.1419266313314438, -2.3409905433654785, 0.6885899901390076, -0.486263245344162, 0.24745675921440125, 0.0773266926407814, -1.4576120376586914, -0.3982316851615906, 0.5072929263114929, 1.5461241006851196, -0.14248280227184296, -0.4542100429534912, -0.7278274297714233, 1.4064719676971436, 0.5960618257522583, -0.18601062893867493, 0.7451125979423523, -0.08698935806751251, 0.9423922896385193, -0.12461002916097641, 0.49670878052711487, -2.7953548431396484, 0.2197953462600708, 1.3046268224716187, 0.40128934383392334, -0.17589667439460754, 0.08781186491250992, -0.3795895278453827, 0.3895346224308014, -0.32309195399284363, -0.6962809562683105, 0.5360589027404785, -1.2265676259994507, -0.04182356595993042, -0.08442018181085587, -0.10123027116060257, 0.253518283367157, -1.0356439352035522, 1.120615839958191, -0.4661605954170227, 0.909939169883728, -0.6232713460922241, 0.26377612352371216, 0.534061074256897, 1.204946756362915, 0.8895784616470337, -0.41770267486572266, -0.33395957946777344, 0.7426832318305969, -0.4146062731742859, 0.7682214379310608, 0.807111918926239, 0.6041626930236816, -0.6690198183059692, -0.25842946767807007, -1.100680947303772, 1.9586819410324097, -0.004758263472467661, -0.1764964759349823, -0.05400367081165314, 0.5927048325538635, -1.27874755859375, -1.177893877029419, 1.1775825023651123, -0.17295967042446136, 0.9154515862464905, -0.3273262083530426, -1.8775712251663208, -0.12321257591247559, -1.6179012060165405, -0.5270315408706665, -0.8541695475578308, -0.5251079797744751, 0.4607515037059784, -1.1275720596313477, 1.2121087312698364, 0.4176100492477417, -2.068014144897461, -0.24706558883190155, -1.586646318435669, 0.7183360457420349, 0.2989931106567383, 0.1659669429063797, -0.7881678938865662, -0.35111790895462036, 1.3295577764511108, -1.2915732860565186, -0.19194243848323822, -1.1369808912277222, -0.3281199336051941, 0.5850030183792114, -0.844680905342102, -0.5581789016723633, -0.2731876075267792, -0.7838815450668335, -1.6797317266464233, -2.0109596252441406, 0.658058762550354, -1.1706775426864624, -0.2991112172603607, 0.1803206503391266, 1.8252850770950317, -0.3615230321884155, 1.2420331239700317, 0.36209937930107117, -1.6360009908676147, 0.09723634272813797, -0.7740774750709534], [-0.3741815984249115, -0.5679690837860107, -1.9060314893722534, -0.15917660295963287, -0.32239922881126404, 0.5265399217605591, -0.5610138177871704, 1.1094056367874146, -0.8470280170440674, 0.41369980573654175, 0.6593052744865417, 0.047689229249954224, 0.5215341448783875, -1.0103703737258911, -0.9887190461158752, 0.8050258159637451, -1.9584908485412598, -0.33900195360183716, -0.49996262788772583, -0.43114200234413147, -0.23585909605026245, -0.11306007951498032, -0.26038461923599243, 1.3310242891311646, 0.20439589023590088, 0.955818235874176, -0.5896473526954651, 0.843145489692688, -0.6169503331184387, -1.1615586280822754, -0.2706218957901001, 1.1971179246902466, 1.3648179769515991, -1.223738431930542, -1.1516131162643433, 1.01773202419281, -1.1323527097702026, -1.7116339206695557, 1.4569673538208008, 0.5053097009658813, 0.5334877967834473, 0.5425856113433838, 1.6424986124038696, -2.7726504802703857, 1.219046950340271, 1.1537232398986816, -0.31108078360557556, -0.20710204541683197, -1.1556892395019531, -2.7770512104034424, 0.6501198410987854, 0.53825443983078, -0.27705731987953186, 0.18855416774749756, 1.7502950429916382, 0.9271353483200073, 1.5387041568756104, -0.04538014531135559, -0.8966932892799377, 0.5195623636245728, 0.26145440340042114, -1.1131088733673096, 2.0641918182373047, 0.6156457662582397, -1.022802472114563, 0.09259095042943954, 1.7365837097167969, -0.9799520373344421, -1.5234167575836182, -2.1805801391601562, 1.2457057237625122, 0.14201894402503967, 0.03538624942302704, -1.0045361518859863, 0.8034607768058777, -0.47127974033355713, -0.6121159195899963, 0.40018993616104126, 0.516309380531311, 0.12084908038377762, 0.3778367042541504, -0.34866610169410706, -0.10367278009653091, 0.3575800657272339, -0.07532544434070587, -0.44401705265045166, -0.138475701212883, -1.4383295774459839, -0.3358980417251587, 1.5064905881881714, 0.36969009041786194, 1.1120413541793823, 0.3638700842857361, -0.9415445923805237, -0.7737846970558167, -0.8311311602592468, -1.1684263944625854, 0.533306896686554, 1.136884331703186, -1.6886407136917114, -0.8425474166870117, 0.7085726857185364, -0.5120551586151123, 0.41842466592788696, 0.10247939825057983, 1.0874695777893066, -0.00880442839115858, -0.45125699043273926, 1.230630874633789, -0.06896205991506577, 0.9533438086509705, 1.042893648147583, -1.351981282234192, -0.05737641453742981, 0.03892437368631363, -0.23152391612529755, -0.4546832740306854, -1.551170825958252, -0.8137653470039368, 1.06755793094635, 1.325490117073059, -1.1467911005020142, -0.08346344530582428, 0.7003480195999146, 2.143639087677002, -0.6034548282623291, -1.1870321035385132, -0.2321334332227707]], [[0.48824548721313477, 0.4659319221973419, -0.6610900163650513, -0.8820440769195557, 0.447788804769516, -0.41589540243148804, 0.13917900621891022, 0.9398158192634583, 1.180398941040039, -0.029025904834270477, 1.6636378765106201, -1.234224796295166, 0.39868658781051636, 0.3710766136646271, 0.2856038510799408, -0.911292314529419, 0.19281357526779175, 0.03569890558719635, 0.25103268027305603, 0.6108710169792175, -0.02372775785624981, 0.6602429151535034, 0.2539234757423401, -0.9637263417243958, 0.15837109088897705, 0.6935259103775024, -1.2894995212554932, -0.04969707131385803, -1.135769009590149, -1.3088386058807373, 1.8054430484771729, 1.4502661228179932, 1.0287195444107056, -0.1883137971162796, -0.45915019512176514, -0.9861394762992859, -1.27089262008667, -0.4569132924079895, 2.1117124557495117, 0.2578282654285431, 0.027568353340029716, -0.17102015018463135, -0.6555578112602234, -0.8010833859443665, 0.043463047593832016, -0.3380880057811737, 0.8175939321517944, -0.8353257179260254, 0.7564360499382019, -0.8819197416305542, -0.30131444334983826, 0.30728042125701904, -1.1031930446624756, 0.6647441983222961, -0.6567326784133911, -1.125853419303894, -1.740280032157898, 0.684039831161499, 0.12656563520431519, -0.5137425661087036, -0.6585550904273987, 1.452391505241394, 1.2247560024261475, 0.33053290843963623, -0.26109668612480164, 0.01175224781036377, -0.6373450756072998, -0.09771153330802917, -0.11110344529151917, 0.847903311252594, 0.9306803345680237, -0.5424461960792542, 1.7597839832305908, 0.17887014150619507, 0.24749180674552917, 1.0813881158828735, 0.6241833567619324, 0.060557786375284195, -0.7277124524116516, -0.8077654242515564, -0.5895281434059143, 2.5395240783691406, -0.7960578203201294, -0.46361419558525085, -1.1564288139343262, -1.1204627752304077, 0.9187869429588318, 0.21371056139469147, 1.210587739944458, 0.5529996752738953, 1.0632151365280151, 0.8844441175460815, -0.2885115444660187, 0.21325565874576569, 1.5135403871536255, -0.072785884141922, 0.8387041687965393, 0.4611741602420807, 0.3990165889263153, -0.5848531723022461, -0.7220320105552673, 0.17740817368030548, -0.3363707959651947, 0.4724200665950775, 0.30919817090034485, -0.3965916633605957, 1.187030553817749, 0.414857417345047, 2.125467300415039, 0.9386197924613953, 0.2828056216239929, -0.6653530597686768, -1.126556634902954, -0.47028475999832153, -0.6564249992370605, -0.26838040351867676, -0.8192300200462341, 1.8635388612747192, -1.7773312330245972, -0.9552298188209534, -1.8794493675231934, -1.5701491832733154, -0.0965300127863884, 1.462016224861145, -1.5632379055023193, -0.055144473910331726, 1.154462218284607, 1.2053030729293823], [-0.682773768901825, -0.3474145829677582, 0.3733212947845459, -1.0774953365325928, -0.10053940117359161, -1.3294799327850342, 0.37908780574798584, -0.724811315536499, 0.2978927493095398, 0.21966932713985443, 0.6924461722373962, -0.7652055621147156, -0.18339085578918457, 0.4301392734050751, -0.42310628294944763, -1.0856612920761108, -0.2800070643424988, -2.0378613471984863, 0.7017877101898193, 1.5226707458496094, 1.7499173879623413, 0.7094577550888062, 0.09203574061393738, -1.012868881225586, -0.1392340511083603, -0.5685219764709473, 0.5855262875556946, -0.7908712029457092, 1.050761342048645, 0.9375513792037964, -1.8739819526672363, 1.9414124488830566, -2.074345827102661, -2.8002612590789795, 0.02363557741045952, 0.4339272677898407, 0.6825065612792969, -0.050442054867744446, 0.6989878416061401, 1.3988279104232788, 0.47906461358070374, 0.6951351761817932, 1.995592713356018, 1.0645623207092285, 1.3607271909713745, 0.1271534413099289, -0.05051971226930618, -0.7852003574371338, -0.04533195123076439, 0.8242767453193665, 0.004542812239378691, 0.30765458941459656, -0.9517903923988342, 0.9481820464134216, 0.6423976421356201, -1.417776346206665, -1.5878087282180786, -0.08273375779390335, 0.2886653542518616, 1.2086002826690674, 0.31878793239593506, -0.5120458006858826, -0.367968887090683, 0.5420756340026855, 0.7622987627983093, -0.7989364862442017, -0.4804041087627411, 0.04638608545064926, 0.31448668241500854, -0.5501675605773926, -0.5558958649635315, -0.20298327505588531, -0.9036465883255005, 0.6470814943313599, 0.08710502833127975, 0.9972400665283203, 0.20340459048748016, -1.0713114738464355, -1.646020531654358, -0.1220318078994751, -1.2099480628967285, 0.8636115789413452, 0.6711100339889526, -0.8580326437950134, -1.2252638339996338, -0.20966202020645142, 0.9155260324478149, -0.504792332649231, 0.7357827425003052, 0.396419882774353, 0.6479707360267639, 0.730351984500885, 1.3758177757263184, -0.5440384745597839, 1.0246981382369995, -0.4497791528701782, 0.5635793805122375, 0.3041640818119049, -0.8573818206787109, -0.04755351319909096, 0.1405838131904602, 0.9517506957054138, 1.338797688484192, -0.8237919807434082, -0.5803254842758179, -0.20755551755428314, 0.9205428957939148, 0.5270096659660339, 0.4569602906703949, 0.6095061898231506, -0.033291950821876526, 0.48906874656677246, -1.9627926349639893, 0.2549650967121124, -0.6436073780059814, 0.9578577280044556, -1.2463135719299316, -0.036288950592279434, 0.7612478733062744, -0.09166177362203598, 0.047913722693920135, -0.27768611907958984, -0.12223190069198608, 2.723853588104248, -2.14186429977417, -0.14506998658180237, -0.5639910101890564, -2.4615533351898193], [-0.9775714874267578, 0.06596802920103073, -2.5828304290771484, 0.13995926082134247, -0.5001170635223389, -0.3619912564754486, -2.331432342529297, 0.849507212638855, 0.05789206176996231, 0.7520462870597839, -1.172677993774414, 1.6461572647094727, 1.1996865272521973, 1.7999931573867798, -1.962032675743103, -0.6379231214523315, 0.24533331394195557, -2.0322158336639404, 1.8885799646377563, -2.199120283126831, -1.0744690895080566, 0.9953995943069458, -0.9019088745117188, 0.16767676174640656, 1.0454705953598022, 1.2755998373031616, 1.1003957986831665, -0.20205187797546387, 0.3944075107574463, 0.44609323143959045, -1.3012267351150513, 0.3670934736728668, -0.3894144296646118, 0.7885136008262634, 1.877168893814087, -0.006362115032970905, 0.1756359487771988, -1.2653526067733765, -0.4331488013267517, 0.02529160864651203, -0.3874112367630005, 0.5032381415367126, 1.657974123954773, 1.0146721601486206, -0.4840880334377289, 0.3445424735546112, 0.8919362425804138, -0.39671969413757324, -0.28009751439094543, 0.8068174719810486, -0.29776594042778015, 0.8220131397247314, 0.09769539535045624, -1.1053502559661865, -0.8423551321029663, -0.036264825612306595, -0.7409461736679077, 0.8694995641708374, -1.992031216621399, 1.0534093379974365, -0.8987080454826355, -0.4593382477760315, 0.6205342411994934, 1.0970323085784912, 0.2959764897823334, -1.0006712675094604, 0.21099452674388885, 0.4699585437774658, -0.6912643313407898, -0.5473299026489258, 0.5991985201835632, -1.9318935871124268, -1.4988480806350708, -1.1010339260101318, 0.20660847425460815, 0.037710342556238174, 1.2114207744598389, -1.6291131973266602, -0.4339240491390228, 0.9282779693603516, 0.14763477444648743, -0.9349811673164368, 1.7056212425231934, 0.1408304125070572, -0.3331157863140106, -0.08136916905641556, 0.8223782181739807, -1.5463923215866089, -1.3850663900375366, 0.525520384311676, 0.5626959204673767, 1.5088785886764526, -0.5066803693771362, 0.623961865901947, -0.4263792335987091, -0.12167128920555115, 1.5405718088150024, -0.22662971913814545, -0.3082825541496277, -0.5691534280776978, -0.36646175384521484, -0.5446569919586182, -1.0370469093322754, 0.08912371098995209, 0.8391822576522827, -0.22894053161144257, 0.1841265708208084, 1.024466872215271, 1.1038130521774292, 0.016012130305171013, 0.6297092437744141, -0.7600225210189819, 0.3189457356929779, 1.9612489938735962, 1.069602608680725, 0.372188925743103, 1.21847403049469, -0.2830580174922943, -1.742716908454895, 1.9751380681991577, 0.47555774450302124, -0.12328625470399857, 0.7249840497970581, -0.27531933784484863, 0.04444604739546776, 0.5980705618858337, -0.1704220026731491, 1.683348536491394], [-1.2640888690948486, 0.592770516872406, -0.22550636529922485, -0.9588107466697693, 0.04872426763176918, 0.6322351098060608, 0.2954907715320587, 0.7946816086769104, -0.44393500685691833, -2.295839309692383, 0.11924566328525543, -1.3671090602874756, 0.6827769875526428, -0.9378632307052612, 0.13368743658065796, 0.3859158754348755, -1.9781842231750488, 1.0740629434585571, 0.22048352658748627, -0.355180561542511, -1.7985877990722656, -3.3165698051452637, -1.0987073183059692, -0.354271799325943, -0.5758565664291382, -0.9784006476402283, -0.84722900390625, 0.6175102591514587, -0.18991269171237946, -0.9007689356803894, 1.1828184127807617, -2.6735007762908936, 1.2531492710113525, -1.1162790060043335, 1.0592344999313354, 0.723003089427948, -0.043504927307367325, 1.4384281635284424, 2.5618457794189453, -0.20308543741703033, -0.9230587482452393, 0.7533373832702637, 0.5074896216392517, -0.5584816932678223, 1.0377745628356934, -0.24951794743537903, 0.8857873678207397, -0.468813955783844, -2.105599880218506, -0.8565067648887634, 1.2411192655563354, -0.2184268683195114, 0.2355724573135376, 0.25416597723960876, -1.5338854789733887, 0.5044450163841248, -1.668143391609192, 0.6811887621879578, -0.02608410455286503, 0.25449034571647644, 0.653820812702179, -2.3487372398376465, 0.9799166917800903, -0.5944237112998962, 0.5999893546104431, 2.110816717147827, 0.25413182377815247, 0.302655428647995, 0.0569918192923069, -0.6525775194168091, 0.23813624680042267, -0.4117383360862732, -2.4540657997131348, -0.5144926309585571, 1.5097776651382446, -0.6510926485061646, -0.5624218583106995, -0.9865196943283081, -0.26683685183525085, 0.07016367465257645, -1.257041335105896, 1.2042644023895264, -0.7726367115974426, -0.4660685062408447, -1.723659873008728, 0.43625280261039734, 0.8997644782066345, 0.9830609560012817, -1.2618296146392822, 0.051899999380111694, -0.4689561724662781, 0.3050311803817749, -0.5111549496650696, -1.2615947723388672, 0.252005934715271, -0.5113388895988464, 0.8291558027267456, 0.5084102153778076, -0.17428021132946014, 0.7462731003761292, 0.5098350048065186, 0.4494236409664154, 0.9815579652786255, 0.6907323598861694, -1.635348916053772, -1.1529089212417603, 0.04007840156555176, 1.6457152366638184, 0.7614225149154663, 0.23018339276313782, 1.166567325592041, 1.4941710233688354, -0.01038343831896782, 0.27050331234931946, 1.4718554019927979, -0.8727962970733643, 1.311381220817566, -1.5533605813980103, -1.6883878707885742, 0.961986780166626, 0.7339921593666077, -0.33558541536331177, -0.016261877492070198, 0.2877415716648102, 0.4074455201625824, -0.7241671085357666, -0.5118051767349243, 0.5401785969734192], [0.4252950847148895, -2.122272491455078, 1.643807053565979, 0.2592804729938507, -1.1922500133514404, -0.5066969990730286, 0.3402945101261139, 0.19043971598148346, 1.8195563554763794, -0.7786588072776794, -2.3257205486297607, -0.1140405461192131, -0.4380033314228058, -1.611292839050293, 0.34195417165756226, 1.0892963409423828, 1.7288031578063965, 0.41413792967796326, -0.26862335205078125, 0.2129679173231125, 1.0319263935089111, 0.7937835454940796, 0.9589576125144958, -0.5224527716636658, -0.6355143189430237, -0.912903368473053, -1.2792974710464478, -0.6171227693557739, -0.369019091129303, -1.522589921951294, -1.0813372135162354, 0.0019217398948967457, 1.014533281326294, -0.7843608856201172, 0.0015546103240922093, -0.3520076870918274, -0.9770970344543457, 0.8631197214126587, -0.941084086894989, 1.3795086145401, 0.051657646894454956, -0.6785448789596558, 1.2577159404754639, -1.3476444482803345, 1.729809045791626, -0.9513279795646667, -0.5497702956199646, -0.2850606143474579, -0.613957941532135, 0.4629802405834198, 0.8978023529052734, 0.5119398832321167, 1.1877727508544922, 0.6024027466773987, 0.44660133123397827, -0.6997022032737732, 0.499830424785614, -0.9256873726844788, 1.3535937070846558, 2.464329719543457, -0.6994510889053345, 0.06790447980165482, -0.33078277111053467, -0.1074226126074791, 1.7024303674697876, -0.09339563548564911, 1.4115370512008667, 0.5205737948417664, -0.35752251744270325, 1.3163833618164062, 0.6278071999549866, 1.0815825462341309, -1.184921145439148, -1.67021644115448, -0.9307212829589844, 0.8239409923553467, -0.3620869815349579, -1.0544389486312866, -0.8940304517745972, 0.8414161205291748, 1.30785071849823, -0.994615375995636, 1.1730908155441284, -0.25102248787879944, 0.17380759119987488, -0.9605923891067505, -1.0367084741592407, -2.142491579055786, 0.6089485287666321, 0.2503703832626343, -1.1504746675491333, -1.1119307279586792, 0.3623065650463104, -1.7428579330444336, -1.1386840343475342, -1.0906517505645752, -0.9698300361633301, -1.2130450010299683, -0.6706194877624512, -0.5042710304260254, -1.4601622819900513, 0.7772431373596191, -1.8885376453399658, -0.7513259053230286, -0.827558696269989, 0.13061968982219696, -0.5866267681121826, 1.8383280038833618, 0.5249453783035278, -0.6377003192901611, -1.2905102968215942, -0.12953563034534454, -0.19777433574199677, 0.7117226719856262, 0.7407868504524231, -2.2403817176818848, -0.4203300476074219, -1.347182035446167, 1.5495600700378418, 0.4481280446052551, -0.8473225235939026, 1.2201839685440063, 1.9658116102218628, 0.11395996809005737, 0.6888405084609985, -1.2631193399429321, 0.9462546706199646, 0.5076455473899841], [1.681764841079712, 1.236619472503662, -0.14016872644424438, -1.469470739364624, 1.1430637836456299, 0.7668861746788025, 0.013711158186197281, -0.37491145730018616, 0.2710791230201721, 0.15253879129886627, 0.47636404633522034, -0.6777876615524292, -0.07213156670331955, 0.04753609374165535, -0.3355793058872223, 0.7872490286827087, -0.14082814753055573, 0.8183931708335876, 0.07498662918806076, -0.9117449522018433, 0.25912541151046753, -1.267591953277588, 0.8127560615539551, 0.3457992672920227, -0.9605620503425598, -0.4383152723312378, -0.5982831120491028, 1.5523011684417725, 1.1228255033493042, -2.0258522033691406, 0.2627086043357849, -1.3795182704925537, -0.4238870441913605, 0.03248802572488785, 0.7473639845848083, 0.4296872615814209, 0.8563470840454102, 0.8277211785316467, -0.9123442769050598, -0.24898850917816162, 0.34516310691833496, -0.7204504013061523, 0.9044311046600342, 0.1754918247461319, -1.1338471174240112, 0.48717743158340454, 1.382426142692566, -0.7364432215690613, 0.9285706281661987, 1.9880313873291016, 1.3348867893218994, -1.1585999727249146, 0.13567613065242767, 0.3636234998703003, 0.8674483299255371, 0.9993301033973694, -0.2399461716413498, 0.21433764696121216, -0.33202648162841797, 1.700488805770874, -2.0566790103912354, -0.3447014093399048, 0.738979697227478, -0.02563752420246601, 1.1218101978302002, 0.030731171369552612, -1.4816972017288208, -0.6069990992546082, -0.6138157844543457, -1.4429274797439575, 1.160391926765442, -0.13724885880947113, -2.1463375091552734, -0.2962566614151001, -0.8732689023017883, 1.305947184562683, -0.6123385429382324, 1.2340823411941528, -0.38299116492271423, 0.11983992904424667, -0.9616537094116211, 0.40095555782318115, 0.2080669105052948, -0.27160561084747314, -0.5776327252388, -0.5665181279182434, -0.5549238324165344, -1.3452751636505127, -1.2440593242645264, 1.8729755878448486, 1.444864273071289, 1.4333419799804688, -0.3887439966201782, -0.4296886920928955, -1.0533838272094727, -0.7251152992248535, 0.4197939932346344, 1.1983412504196167, -0.9123865962028503, -0.5450230836868286, 0.05493619665503502, 0.3938523232936859, 2.2182462215423584, -1.1686040163040161, 1.0997064113616943, -1.4014384746551514, 0.19694563746452332, -0.4486280679702759, -0.23115777969360352, 1.3640691041946411, -0.285245418548584, 0.9067596197128296, -0.5406053066253662, -0.945216953754425, 0.6559333801269531, -0.4155402183532715, 1.4637166261672974, 1.774620532989502, 1.2523802518844604, 0.10972582548856735, -0.3339826166629791, -0.14083822071552277, 1.1225000619888306, 1.5438473224639893, 0.25691527128219604, -0.0646119937300682, -0.5085747838020325, -0.8746673464775085], [0.48121392726898193, 0.36211615800857544, -1.0079680681228638, -0.2529520094394684, 0.45910724997520447, -1.0137174129486084, 0.8396899700164795, -1.34429931640625, -0.9321193695068359, 0.4500865638256073, 0.6055466532707214, -1.4114017486572266, -0.4338487684726715, -0.572304368019104, -0.6295216679573059, 1.279121994972229, 0.0537848137319088, 1.1565396785736084, -1.188528060913086, -1.0865304470062256, 0.30443161725997925, 1.0218359231948853, 0.3629685342311859, -0.6292917132377625, 0.8286380767822266, 0.0038390120025724173, -0.15855282545089722, -1.7076526880264282, 1.1461784839630127, -0.6502224206924438, 0.8215570449829102, 1.9056898355484009, 0.35365554690361023, -0.12740753591060638, -0.55108243227005, -0.8373711705207825, -0.4929465651512146, 0.7507971525192261, -0.9628084301948547, 0.8093810081481934, -0.8424029350280762, -0.07108894735574722, 0.20301364362239838, -0.16006413102149963, 0.7874370217323303, -0.41728970408439636, -0.5193379521369934, 0.6246967315673828, -0.2448021024465561, 1.629906415939331, 0.02339125983417034, 0.09752323478460312, -1.9748092889785767, 0.8650099635124207, 1.154707908630371, -0.8805267810821533, 0.8506590723991394, 0.6485615968704224, 1.327346682548523, -0.8775690197944641, 0.11599527299404144, -0.7034357190132141, 0.7068794965744019, 1.034069299697876, -1.437671184539795, 0.40957239270210266, -0.6062510013580322, -1.646881341934204, -0.8274528980255127, 0.2807406783103943, 0.5581746697425842, -0.001768197980709374, 0.3743273615837097, -0.5020667910575867, -1.3996694087982178, 0.1731683909893036, -0.8016153573989868, -0.5809968709945679, -1.2876330614089966, -0.3479958176612854, 0.7847592830657959, -0.5943418741226196, -0.0697663351893425, 0.7229665517807007, -0.008989551104605198, -0.6505613923072815, -1.4968831539154053, -0.8185045123100281, 0.7768266201019287, 1.6843786239624023, -0.9010563492774963, 0.2890993654727936, 1.4170739650726318, -0.6457622647285461, -0.4802156388759613, 0.32129615545272827, -1.2249311208724976, 1.334153652191162, -0.1325421929359436, -0.35518550872802734, 0.39349645376205444, 0.8894724249839783, -0.037938617169857025, -0.2922062873840332, -0.07390042394399643, 1.2539876699447632, 1.1991232633590698, -0.9556299448013306, -1.1791642904281616, 1.3905997276306152, -0.974826991558075, -1.7743942737579346, -0.5531065464019775, -0.6706504821777344, -0.30619367957115173, 0.04031825065612793, 0.911054790019989, -0.10917439311742783, -1.3165885210037231, -0.5439055562019348, 0.03549062833189964, 1.59751558303833, 0.652999997138977, 0.30797091126441956, -3.3446097373962402, 0.20997506380081177, 0.6149776577949524, 0.15846563875675201], [2.1137938499450684, 0.5443645715713501, 0.1417715698480606, 1.4548835754394531, 0.9271689653396606, -0.47087767720222473, 1.0037753582000732, -1.7456244230270386, 1.0287554264068604, -2.2894468307495117, 0.3004337549209595, -0.06436135619878769, 1.4126044511795044, -0.6402176022529602, 0.033625874668359756, 1.3144073486328125, 2.9742586612701416, 0.5840757489204407, 2.988129138946533, 1.0381609201431274, 0.4742748737335205, 1.0323011875152588, -1.0972951650619507, -0.8582924008369446, 0.3505847454071045, -0.3480282425880432, -0.3476833999156952, 0.17333325743675232, -0.39714884757995605, 0.5696738958358765, -0.1732030063867569, 0.6522485613822937, -0.011375603266060352, 0.3885839879512787, 0.3499261736869812, 0.9819582104682922, -0.05468667671084404, 1.8822693824768066, 0.16997560858726501, -1.6961438655853271, -0.6911133527755737, 1.3490140438079834, -0.8304678797721863, 2.502279758453369, -1.1440651416778564, -0.4645979702472687, 1.2012255191802979, -0.35844582319259644, -0.283856064081192, 0.15309616923332214, 0.5952572226524353, -2.2571232318878174, -0.8066494464874268, 1.457108497619629, 1.5100606679916382, 0.9934412837028503, -0.3255525827407837, -2.6813085079193115, -0.8520948886871338, -0.7366587519645691, 0.150748610496521, 0.5094485282897949, 1.160516381263733, -0.8602792620658875, 1.3651158809661865, -0.7651495933532715, -1.7422951459884644, 1.0200984477996826, -0.6480106711387634, -0.46050119400024414, -0.729358434677124, -0.027219828218221664, -1.9297219514846802, -1.0241031646728516, -0.11539846658706665, -1.3130549192428589, 0.05414947122335434, 1.3587419986724854, -0.44401586055755615, -0.43242838978767395, -0.4857322573661804, -0.0860581248998642, 1.0489505529403687, 1.1664633750915527, -0.13299676775932312, -3.0896475315093994, 0.9520403146743774, 0.46537888050079346, 0.9785229563713074, 0.7997164726257324, 0.14357860386371613, -1.1343817710876465, -0.0721852034330368, -0.7532082796096802, 0.6394829154014587, 1.1355432271957397, -0.04321036860346794, -0.18685659766197205, 0.43456339836120605, -0.4122484624385834, 0.797776460647583, 0.11175544559955597, 0.8684425354003906, -0.4433973729610443, -0.8849586248397827, -1.1382060050964355, -0.6514495611190796, -0.8490086793899536, 0.016140522435307503, -0.7117472290992737, -1.767126441001892, -0.6012815833091736, 0.1198711097240448, -1.3825324773788452, -0.6061962842941284, 1.8044885396957397, 0.66558837890625, -1.0225226879119873, -0.7588401436805725, 1.3845640420913696, -1.7311367988586426, -0.09462463110685349, 0.3019554018974304, -2.4655301570892334, -0.8921384811401367, 0.9594011306762695, 0.3366337716579437, 1.3272409439086914], [0.7197177410125732, 0.4836660325527191, 1.0893683433532715, 0.30089738965034485, 0.42596307396888733, -0.10192219913005829, 0.2484753131866455, 0.3707464635372162, -1.0070209503173828, -0.5778390765190125, -0.7866711616516113, -0.6963123083114624, -1.1708122491836548, 0.26715636253356934, -0.9519907832145691, 0.1763426661491394, -0.46837112307548523, 0.31767114996910095, 0.22249151766300201, 1.150386929512024, -1.0035762786865234, 1.7481459379196167, -1.369645595550537, 1.2642086744308472, 0.9092826843261719, -0.3213339149951935, -0.4382779598236084, 0.4901169240474701, -0.9809914827346802, -0.6808668375015259, 0.37682172656059265, -0.14535507559776306, 0.1830260008573532, 1.4579873085021973, -0.4165448248386383, -0.7183515429496765, -0.3059110939502716, -0.3346541225910187, 0.6962003111839294, 0.887997031211853, 0.299269437789917, -1.325836181640625, 0.2937941551208496, -0.4129321575164795, 2.4995944499969482, -0.7726625204086304, 0.1573077142238617, 0.3386736512184143, 1.2450125217437744, -0.5269407629966736, 0.9302899241447449, 1.235999345779419, 2.3886873722076416, 0.8585778474807739, -1.4773707389831543, 0.8018807172775269, -0.7535738348960876, 0.8731505274772644, -0.1094900444149971, -1.0916327238082886, -2.3301761150360107, -2.043858051300049, 1.6486830711364746, -1.4335967302322388, 1.9288164377212524, -1.545264720916748, -0.06975317746400833, 1.0783405303955078, -2.502031087875366, -1.7071559429168701, 1.3733758926391602, 0.5969521999359131, -0.09006864577531815, -0.7280538082122803, -1.2398616075515747, -0.4592927396297455, -0.6583689451217651, 0.2987477481365204, -0.39202505350112915, -0.6359869837760925, -0.45841220021247864, 1.0749030113220215, -0.9003161787986755, -1.806469440460205, -0.49896299839019775, -0.16045577824115753, -0.6426881551742554, -0.05975012853741646, -1.7657543420791626, -1.0864242315292358, 0.49881574511528015, 1.377691626548767, 0.15132185816764832, -1.3075683116912842, -0.7405800819396973, -0.2259967029094696, -0.5343083143234253, 0.46100717782974243, -0.8748522400856018, -0.5864462852478027, -0.4576623737812042, 0.1324492245912552, 2.0766921043395996, 0.7581313848495483, -0.37252277135849, 0.9589493274688721, 0.0012025191681459546, 0.06765951961278915, -0.4588135778903961, -0.5355366468429565, 1.448952555656433, 0.9065912365913391, 1.4760743379592896, 0.3960496783256531, 0.41777661442756653, -0.9578048586845398, 0.8848563432693481, -0.20224469900131226, -0.0983411893248558, 0.39850279688835144, 1.0851936340332031, -0.39192941784858704, 0.4532259702682495, -0.7192601561546326, -0.46335744857788086, -0.5568006634712219, -0.25053203105926514, -0.41827765107154846], [0.5513116121292114, -0.34311550855636597, 0.454925537109375, -3.009071111679077, 1.81269109249115, 0.3305357098579407, 1.0122709274291992, -2.0695483684539795, -0.05436908081173897, -0.17439870536327362, 0.49956777691841125, -0.19616755843162537, -1.0582078695297241, 0.8359677195549011, -0.3054090142250061, -0.6318855881690979, 0.8699772357940674, -2.1537296772003174, 0.02737155742943287, 2.8649508953094482, 0.3898743987083435, 1.1885615587234497, -1.2384127378463745, 1.6447384357452393, -0.37053316831588745, -1.1396108865737915, 0.6837661862373352, -1.0434626340866089, -1.2132130861282349, 0.6900500655174255, 1.8824853897094727, -0.7778014540672302, -1.579858660697937, -1.8806748390197754, -0.15393628180027008, 1.2737147808074951, -0.07098446041345596, 0.6999719738960266, 1.553226351737976, 0.48685625195503235, 0.5583099126815796, 0.5695085525512695, 1.6365669965744019, -0.09731185436248779, -1.6535868644714355, -0.21871598064899445, -1.7670773267745972, -0.776766836643219, 0.4997209310531616, 1.0433368682861328, 0.8975602984428406, 0.8193880319595337, 0.25258296728134155, -1.7455437183380127, -1.1306023597717285, 0.05762040615081787, 0.4916660785675049, -0.4129939675331116, -0.9472148418426514, -1.4078022241592407, -0.5397443771362305, 0.97725909948349, 0.3028455078601837, -0.8495382070541382, 0.5636820793151855, -1.7689229249954224, -1.2801302671432495, 0.07321271300315857, -0.3476465940475464, -0.7781209349632263, -1.3160886764526367, -0.4422365128993988, -0.902190625667572, 0.37323063611984253, -0.6706060767173767, -1.7524237632751465, 0.8744760751724243, 0.38277924060821533, -0.0008601280278526247, 0.7349449992179871, -0.226084902882576, -0.9631257653236389, -1.0338070392608643, -0.609662652015686, 0.6876091361045837, -1.3135697841644287, -0.9058085680007935, 0.20528042316436768, 1.4675558805465698, 0.7588335871696472, 0.6286939978599548, 0.5612419843673706, -0.0060464125126600266, -1.226691722869873, -1.201554298400879, -0.20750531554222107, -0.8866971135139465, -0.0047117238864302635, -1.1998008489608765, -0.7931584715843201, 0.5708068013191223, 0.14507479965686798, -0.4291701316833496, 0.5953295826911926, 1.7563790082931519, -0.5024926066398621, 1.606939673423767, -0.8828917145729065, 3.259094715118408, 0.5569793581962585, 0.32052335143089294, 0.580987811088562, -0.20067554712295532, 1.644472599029541, 0.6443161368370056, -1.5130339860916138, 1.4682950973510742, -0.09542939811944962, 0.6358928680419922, -0.4155994951725006, -0.8044556379318237, 0.054522011429071426, 0.10347051173448563, 0.6222780346870422, 0.05388018861413002, 0.6205618977546692, -0.6949456930160522, 0.6292359232902527], [0.7165240049362183, 0.8523984551429749, -0.4966700077056885, 0.2879338562488556, 1.6597164869308472, -1.2051962614059448, 1.1268748044967651, -1.6892647743225098, -1.2182918787002563, 0.7297524213790894, -0.14900323748588562, 0.5181775689125061, -0.28073614835739136, 0.7357675433158875, -0.5477038025856018, -0.21678388118743896, 0.8938502669334412, -0.35269322991371155, -0.2306891530752182, 0.360871821641922, 0.5039281845092773, 1.0520738363265991, 0.6863889694213867, -0.004257735796272755, -1.1077244281768799, 0.9043036103248596, 0.5248858332633972, 0.20120996236801147, 0.8375809192657471, -0.06441818922758102, 0.9143885970115662, -1.8221042156219482, 1.0905005931854248, 1.069994330406189, 1.4426109790802002, 0.4684544801712036, -1.0874303579330444, 0.5681594014167786, -1.5036836862564087, -0.9556517601013184, -0.6236855387687683, -1.6614161729812622, -1.1000787019729614, -0.040823791176080704, -0.827062726020813, -0.29748600721359253, 0.16314007341861725, 1.3090929985046387, -0.09196338057518005, -2.187732696533203, -0.7803207635879517, 0.10117601603269577, -0.457685261964798, -2.379783868789673, -1.3808114528656006, 0.9690495133399963, -0.24941663444042206, -0.522703230381012, -1.6645087003707886, 1.2268174886703491, 1.0577908754348755, -0.22838617861270905, -0.8842016458511353, 0.0634426698088646, -0.697750449180603, -1.5774781703948975, -0.36958229541778564, -0.21688610315322876, -1.589718222618103, 1.7764889001846313, -0.9341768622398376, 2.231510877609253, -1.4613384008407593, -0.37756067514419556, -1.5330212116241455, 1.203202247619629, 0.3198357820510864, 0.1426030695438385, 0.91164630651474, -1.374449610710144, -0.059582144021987915, 0.17186763882637024, 1.3811968564987183, -0.0891047939658165, 0.450074166059494, -1.8143895864486694, -0.22521373629570007, 2.0543272495269775, 1.064313530921936, -0.9514193534851074, -0.19199158251285553, 0.16209670901298523, -0.3605574667453766, 0.641353189945221, 1.1477068662643433, 0.28941118717193604, 1.3364605903625488, 0.9069250226020813, 0.8026208281517029, 0.05006938800215721, 0.7496051788330078, 0.5103738307952881, 0.9452590942382812, -0.902428388595581, 0.3812578618526459, -0.43009650707244873, -1.810728669166565, -0.7279929518699646, -1.0188387632369995, 0.9937520623207092, -0.48462820053100586, 1.2616419792175293, -0.627020537853241, -0.21312136948108673, -1.2188541889190674, -0.646919846534729, -0.7938439249992371, 0.7272856831550598, 0.13757453858852386, 0.9166785478591919, 0.5536134243011475, 0.07271024584770203, -1.8426145315170288, 0.530480682849884, -0.8895980715751648, 0.23327277600765228, -0.4727219343185425, 0.4152953028678894], [1.2565666437149048, -0.05021514743566513, -0.24012966454029083, -1.011159062385559, -0.2230914980173111, 1.0443434715270996, -0.24184450507164001, -0.1349116414785385, -0.2206844985485077, -0.7653365135192871, 2.0755600929260254, -0.16438719630241394, -0.2848852872848511, -0.6646531820297241, 0.29758089780807495, -1.7634190320968628, 2.1756391525268555, -1.0801732540130615, -1.2356600761413574, 0.613564670085907, 1.2360284328460693, -0.2126358449459076, -1.468295931816101, -0.8078775405883789, -0.48638594150543213, -1.3196196556091309, 0.4861059784889221, 0.14383843541145325, 0.514939546585083, 1.063650369644165, 0.1577034443616867, 0.02289360761642456, 0.40943655371665955, 1.6138986349105835, -0.0016965785762295127, 0.09283289313316345, -1.1717127561569214, 2.753209352493286, -0.8186193108558655, 0.8169022798538208, 0.09529778361320496, -0.6883029937744141, -0.19318373501300812, -0.4474935531616211, -0.019391098991036415, 0.020417556166648865, -2.455949306488037, -1.8005238771438599, -1.588599681854248, 1.0904711484909058, 2.5525896549224854, -0.279909610748291, 0.8693670630455017, -0.5293795466423035, -1.4129225015640259, 0.7998665571212769, 2.687095880508423, 1.2922784090042114, -1.709909200668335, 0.7718319296836853, -0.1948351263999939, 0.635724663734436, -1.9667601585388184, 0.14886939525604248, -0.9028909802436829, 0.3962942957878113, -0.12202100455760956, 0.509881317615509, 2.099290609359741, 1.8067271709442139, -0.41555607318878174, 1.649251937866211, 1.84408700466156, 0.99382483959198, 1.5198982954025269, 0.29303911328315735, 0.30313464999198914, -0.29119786620140076, 0.5562863349914551, 0.002967487322166562, 0.5062218904495239, 1.2917215824127197, -0.5515186786651611, 0.8573994636535645, -1.2851288318634033, 0.3555039167404175, -0.9204543828964233, 0.6284090876579285, -0.20309677720069885, 0.8138818740844727, 0.04773574694991112, 0.6023393273353577, 0.9030269384384155, 0.2644571363925934, 1.1296027898788452, -1.7357923984527588, 0.8743008375167847, 0.635758101940155, -0.1531345099210739, -0.1530989110469818, 0.6228098273277283, 2.025336265563965, -1.3229855298995972, 0.36886805295944214, -0.048588965088129044, 0.237392395734787, 0.33427560329437256, -1.3321810960769653, 0.46832704544067383, -0.4633623957633972, -0.8915895223617554, -1.7216086387634277, -0.07581961154937744, -0.6168731451034546, -1.2195632457733154, -0.7194733023643494, 1.161586046218872, 0.7292931079864502, -0.6090444326400757, -1.3107913732528687, 0.7227715849876404, 0.3722756505012512, -0.9457376599311829, 0.8359237909317017, 1.8251334428787231, 0.14448963105678558, -0.11872459948062897, 0.4334177076816559], [-0.31264668703079224, 0.8337794542312622, 0.6516939401626587, -0.7117243409156799, 1.1567435264587402, -0.6850190758705139, -1.4613831043243408, 0.04087549075484276, -0.22026407718658447, 1.3702043294906616, 0.6946980357170105, 0.5890924334526062, 0.06871679425239563, -0.02700357511639595, -0.43281829357147217, 0.017327453941106796, -1.1887954473495483, 1.6390918493270874, 0.031095951795578003, -1.7171282768249512, -0.9493089914321899, 1.7760138511657715, -1.517008900642395, 0.019463863223791122, -0.10226259380578995, -0.07935500144958496, -0.5927097797393799, 0.2760958671569824, 2.6789276599884033, -0.3392663896083832, 0.24300211668014526, -1.1838712692260742, 1.722959280014038, 0.7643702030181885, 0.22661332786083221, 1.3993045091629028, 0.024618074297904968, -1.221771478652954, -1.3654361963272095, 0.8027392625808716, 0.003875387366861105, -0.2410724014043808, 1.1382033824920654, -0.5281682014465332, -0.05027325451374054, -0.37216973304748535, 0.4086242616176605, -0.0034849222283810377, 1.4904934167861938, 0.5868389010429382, 0.855829119682312, -1.0201539993286133, -0.29577022790908813, 0.38263410329818726, 0.9871318340301514, 0.8180399537086487, 0.9030272960662842, -1.0891761779785156, 0.19704636931419373, 2.741323232650757, -1.8277417421340942, -1.2326358556747437, -0.7087929844856262, -0.06056235358119011, -0.11728871613740921, -0.4212873876094818, 1.749493956565857, 0.7464087009429932, -0.4118262231349945, -0.011436866596341133, 0.2849067151546478, -0.2906261384487152, 1.5454630851745605, -0.7162644863128662, 0.8162169456481934, -0.7231735587120056, 1.467903971672058, 1.6813441514968872, -0.09141655266284943, 0.882964015007019, 0.17029248178005219, -1.1126775741577148, 0.6793035864830017, 0.8975088596343994, -0.9571933746337891, -1.2776974439620972, 0.9470950961112976, -0.8401296734809875, -0.2711552083492279, 0.37284156680107117, 0.9059924483299255, 0.06480156630277634, -1.366951584815979, 0.3487396240234375, -1.1707111597061157, 0.7950973510742188, 0.4384045898914337, -2.012402296066284, 1.781374216079712, -0.3071358799934387, 1.1258612871170044, 0.3282966613769531, -2.164771556854248, -0.556953489780426, -2.3527822494506836, -0.8186056017875671, 0.9749566316604614, 0.8021014928817749, 0.0654020607471466, -0.13767927885055542, 0.3273935317993164, -0.33037707209587097, -0.4505251944065094, 0.24486880004405975, -0.39430075883865356, 1.6815968751907349, -2.1321234703063965, 0.34406808018684387, -0.007894616574048996, -1.7844061851501465, 1.1516170501708984, -0.17306314408779144, 1.1947169303894043, 0.8434891104698181, -0.6049835085868835, 1.1428759098052979, 0.9625581502914429, -0.13176405429840088], [-1.537039875984192, -1.7796927690505981, -1.1641861200332642, 0.08274312317371368, 0.2948397696018219, -0.4807668626308441, -1.673654317855835, -0.8355907797813416, 0.6650204062461853, 1.0515767335891724, 1.7449814081192017, -1.4548989534378052, -0.690495491027832, 0.41347235441207886, -1.6792359352111816, 0.15717318654060364, -0.8221303820610046, 0.25388118624687195, 0.0820305198431015, 1.2293137311935425, 0.2530927360057831, -0.28191259503364563, -0.17619648575782776, 1.6219605207443237, -0.6884583830833435, -0.1229783296585083, 0.49193090200424194, -2.0146114826202393, -0.42755573987960815, -0.8491193652153015, -0.4409312307834625, 0.2743689715862274, 0.9465736746788025, -0.723851203918457, -3.3563430309295654, -0.3160346448421478, -0.3658709228038788, 0.1354735791683197, -1.4282503128051758, -1.501316785812378, -0.33452755212783813, 0.5797657370567322, 0.6266540288925171, -1.5873554944992065, -0.940228283405304, 0.20358264446258545, -0.1253138780593872, 2.0049850940704346, 0.645361065864563, -0.08941902220249176, 0.5802727341651917, 0.3719709813594818, 0.22333084046840668, -1.7667261362075806, -1.4706857204437256, 0.056662824004888535, 1.413405179977417, 1.7021719217300415, 2.2015554904937744, 1.483197569847107, 0.5192873477935791, 0.8472614884376526, 1.3111941814422607, -0.44606029987335205, -1.0867146253585815, 0.9708684086799622, 1.8436883687973022, -0.8621194958686829, 0.2051762342453003, 1.3086702823638916, 0.24993181228637695, 0.925814688205719, 0.30250588059425354, 0.7440270781517029, 0.8039475083351135, 1.3641908168792725, -0.5043622851371765, -0.11639468371868134, -0.8977652788162231, 0.8112891912460327, 1.8567326068878174, 0.4027596712112427, 0.35342252254486084, 0.42651793360710144, 0.17590674757957458, 0.32906872034072876, -0.9412386417388916, 0.4746551811695099, -0.31834128499031067, -1.547900676727295, 0.21175295114517212, -0.4791608452796936, -0.6537463068962097, -0.6554077863693237, 1.0507023334503174, -0.5374423265457153, 0.5745771527290344, -0.021781517192721367, 1.1139649152755737, -0.63852459192276, -1.1206302642822266, 1.0315420627593994, 1.2468085289001465, 0.5290427803993225, 2.094788074493408, -0.8480331301689148, -0.61360102891922, 0.6408973336219788, -1.2003921270370483, 0.4238766133785248, 0.6498985290527344, -0.02999155968427658, 1.8380022048950195, -0.3190518319606781, -0.16881424188613892, 0.6106098890304565, 2.382627248764038, 1.2276891469955444, 1.482829213142395, -0.7204172015190125, 0.5443226099014282, -0.8115397691726685, -1.4812681674957275, 0.9101402759552002, -0.020704232156276703, -0.16622862219810486, -1.8133796453475952, 1.0504536628723145], [-0.9872454404830933, -1.332004427909851, -0.31829336285591125, 0.6768736243247986, 0.7756835222244263, 0.8964015245437622, -1.5719836950302124, 0.7237978577613831, -0.31758975982666016, 1.2084105014801025, -0.3455631136894226, -0.5536565184593201, 0.08109553158283234, 0.10867836326360703, -0.6544276475906372, 1.3810784816741943, -0.2347581833600998, -0.14644336700439453, -0.8608856201171875, -0.9963887333869934, -0.41591402888298035, 0.23614361882209778, 0.08691921830177307, -1.7269015312194824, -1.5137964487075806, -0.7253245115280151, -0.6688525080680847, 0.3170425593852997, -1.7608611583709717, 0.1656782627105713, -0.23409678041934967, 0.8116233348846436, 0.6052559614181519, 0.03842862695455551, -0.34719231724739075, 1.2132229804992676, -0.8402060866355896, -0.24758513271808624, -2.1051411628723145, 0.569145917892456, 0.9635390639305115, -0.6112846732139587, 1.5126800537109375, -1.9040207862854004, -0.23856276273727417, -0.2933826446533203, 1.7734050750732422, -1.1255849599838257, -0.5038167238235474, 0.967627227306366, -0.5478131175041199, 1.8726866245269775, -1.2036283016204834, -0.42050349712371826, -0.16708973050117493, 1.487804651260376, 2.094770669937134, -0.972628653049469, -1.0179544687271118, 1.2006126642227173, -0.8291254639625549, -0.824219286441803, 0.5292119979858398, 0.3037181794643402, 0.3464793860912323, -1.7640422582626343, 0.7843592166900635, -0.41908615827560425, -0.10935933142900467, 0.43130749464035034, -1.3516777753829956, 1.8519797325134277, -0.3909153342247009, 1.0422271490097046, 1.3147990703582764, -1.175359845161438, 1.4846534729003906, -1.1025948524475098, 0.281324177980423, 2.5804901123046875, -1.7077264785766602, -1.7824984788894653, 1.9567493200302124, 0.17470228672027588, 0.6590259075164795, 0.6856826543807983, -0.10878875851631165, -0.4721776247024536, 1.2135595083236694, 0.2722812592983246, 0.5377423167228699, 0.11426739394664764, -0.6577286720275879, 0.30582207441329956, 0.09119964390993118, 0.4875304400920868, 0.9734408855438232, -0.8888710141181946, 0.712390661239624, 0.4336893558502197, 0.1415388584136963, -1.0118249654769897, -0.47493812441825867, 0.13280075788497925, -0.8712392449378967, 0.39557045698165894, -0.616009533405304, 0.12976515293121338, -1.7311981916427612, -0.8485095500946045, -3.28653621673584, 1.7842885255813599, -0.9401823878288269, -0.17323215305805206, -1.0001260042190552, -1.101996660232544, 0.4120391607284546, -0.3610392212867737, -1.0349282026290894, 0.8904671669006348, -1.8013181686401367, -0.2947675585746765, 0.4832351803779602, -0.9799253940582275, -0.3902130722999573, 0.21949949860572815, 0.7674292325973511, 1.4867045879364014], [0.7745168805122375, -0.9023313522338867, -0.7837731838226318, -0.9074510335922241, 0.04484066367149353, 0.1048363521695137, 0.993967592716217, -2.028278350830078, -0.19499875605106354, -0.5683147311210632, -1.2677452564239502, -0.22721543908119202, 1.0517417192459106, 0.08460526913404465, -0.7904108166694641, -1.2932114601135254, -0.6627331972122192, -0.3480501174926758, -1.6239616870880127, -0.5902851819992065, 1.7002193927764893, 0.9722424149513245, 0.3923276960849762, 0.6416928768157959, -0.48159661889076233, 1.890053391456604, -2.1920595169067383, 0.18018649518489838, 0.32461461424827576, -0.9412063360214233, 0.7128437161445618, -0.42075115442276, 0.5028529763221741, -0.46648266911506653, 0.0019571532029658556, -0.5485085844993591, -0.44764620065689087, 1.038842797279358, -1.6598187685012817, -1.1935505867004395, 1.18582284450531, -1.4007092714309692, -0.016054103150963783, 0.32576826214790344, -0.3860907554626465, -1.2800099849700928, -0.7116096019744873, -0.22796596586704254, 0.7612624168395996, -0.7820675373077393, -1.111521601676941, 0.5487907528877258, -0.7519760727882385, -1.9823260307312012, 0.1254272609949112, 0.40483468770980835, 1.4349119663238525, -0.4380383789539337, 1.1394398212432861, -0.07154589891433716, 1.825844407081604, 0.30546507239341736, -1.8021519184112549, 2.11336088180542, 0.05381692200899124, -0.9818361401557922, -1.1495599746704102, -1.340733528137207, 2.001858949661255, -3.363964319229126, 0.5306057929992676, 0.43653491139411926, -0.7117525935173035, 0.7746977806091309, 0.3403245210647583, 2.078564405441284, -0.3241167962551117, 0.6789897084236145, 0.060454998165369034, -1.5316500663757324, -1.1685960292816162, -0.37978506088256836, 0.4148605763912201, -0.5852217078208923, -0.4136850833892822, -0.026823366060853004, -0.4151172339916229, 0.16045726835727692, 0.701420247554779, 0.4770796597003937, 0.8798915147781372, -0.879492998123169, 0.5971494317054749, 0.6481996774673462, -1.21241295337677, -0.5161004066467285, -0.2517854571342468, -0.9402661323547363, 1.9986951351165771, -0.9868354201316833, 0.2865910530090332, -0.5017353892326355, 0.08744952082633972, 1.837022066116333, -0.6963350176811218, 0.43549296259880066, -0.1892227828502655, 1.3386752605438232, -0.015057873912155628, 0.5604950785636902, 0.4000062346458435, 1.0603644847869873, 2.003031015396118, 1.0390751361846924, -1.2271943092346191, -0.8993044495582581, -0.2672240734100342, -0.49252450466156006, 0.30727618932724, 1.5829704999923706, 0.6931464672088623, 0.04256065934896469, -1.7651060819625854, -0.3429046869277954, -0.5159236192703247, -0.23920132219791412, -2.3110461235046387, 0.5746459364891052], [-0.04790869355201721, -0.14990967512130737, -0.36104318499565125, 1.0086853504180908, 0.4109744727611542, 0.1463862806558609, 1.0914323329925537, 0.7946382761001587, -1.1843739748001099, 0.45665308833122253, 1.5811610221862793, 0.8419339656829834, 0.743827760219574, -0.6922826766967773, 0.9269388914108276, -1.925373911857605, -0.29327720403671265, -0.07031508535146713, 0.36446937918663025, -0.040046725422143936, -0.05824674293398857, -0.8137903809547424, 0.6605480313301086, 0.5154947638511658, -0.6465844511985779, 0.27764660120010376, 0.38279691338539124, -0.7475467324256897, -0.5801173448562622, -1.413084626197815, -9.194903213938233e-06, -0.5182446837425232, 0.250660240650177, -0.5914282202720642, -1.3350307941436768, -0.3012944757938385, -0.800622820854187, -1.3849101066589355, -0.5066794753074646, -0.6877947449684143, 1.4143767356872559, -1.4819536209106445, 0.6278700232505798, 1.413347840309143, 0.9044667482376099, -0.9041612148284912, -1.2782063484191895, -1.1083076000213623, -0.6341636776924133, -2.4279966354370117, -0.3992476761341095, -0.3933698832988739, -0.3930785059928894, -1.3128994703292847, -0.031703148037195206, -1.2886428833007812, 2.3115427494049072, -0.6119480133056641, -0.35367098450660706, -0.4775497615337372, -1.20103120803833, 0.6835257411003113, 1.1498256921768188, 0.14349038898944855, 2.0116231441497803, 1.4916938543319702, 0.2777058780193329, 0.42550399899482727, 0.4425378143787384, -1.2121912240982056, -0.7114737629890442, -0.5393263101577759, -0.5966299772262573, 1.4133515357971191, 0.8567023873329163, -1.0193934440612793, 1.5437263250350952, 0.4025858938694, -1.7832443714141846, 3.153712272644043, -0.4676871597766876, -0.8576942682266235, 1.2243188619613647, -0.27955347299575806, 0.05743815377354622, -1.5057333707809448, -0.8681856989860535, -1.272728681564331, -0.38270217180252075, 1.0369179248809814, 0.9413869976997375, 0.5893739461898804, 0.8905807733535767, -1.3456693887710571, -0.8047630786895752, 0.21136000752449036, -1.2180367708206177, -0.47501248121261597, 0.5933789610862732, 0.3587406277656555, -1.1848551034927368, -0.9279890060424805, 0.4000551104545593, -0.2687928378582001, -0.96356201171875, 0.7888712882995605, -0.7970948219299316, -0.030697694048285484, -0.744381308555603, -0.6090039610862732, 1.8347686529159546, 1.7568111419677734, -0.7957988977432251, 1.1292603015899658, -0.2839817702770233, -1.4591258764266968, 0.5675930380821228, 0.8020975589752197, 0.2863004207611084, 0.3687734007835388, 0.9693037867546082, 0.2772623896598816, 1.7656162977218628, 0.8158847689628601, -0.6088916063308716, -0.16843575239181519, 0.819179892539978, -0.7877398729324341], [0.584380567073822, 1.714127779006958, -0.5808266401290894, -0.6616004109382629, 0.3029765784740448, 0.4232144057750702, -1.1938360929489136, -0.5747410655021667, 0.23183177411556244, -0.7215277552604675, 0.7912909388542175, 0.05547093227505684, -0.7788790464401245, 1.8493337631225586, 1.9151066541671753, -0.6882320642471313, 1.835187315940857, 0.005115824285894632, 1.735782504081726, -0.9862898588180542, -0.34372609853744507, 0.6805189251899719, 0.8289587497711182, -0.18105991184711456, 0.3130098879337311, 0.343403160572052, 0.5181646943092346, -0.1007615327835083, -1.1044948101043701, -1.9688746929168701, 0.6588712930679321, -0.05249454081058502, 0.6116201877593994, -0.4340014159679413, 0.11977685242891312, -2.286587953567505, 0.11436706781387329, 0.6768357753753662, -0.09072252362966537, -0.6833965182304382, 1.286714792251587, -0.10638095438480377, 0.02183588221669197, 0.021533340215682983, -0.778021514415741, -0.07661886513233185, -0.7388319373130798, -1.5373151302337646, -0.1513594686985016, -1.2710703611373901, 0.5709294676780701, 1.389743447303772, -0.05896029621362686, -0.7673498392105103, 0.7589558959007263, -2.988382339477539, 1.0967717170715332, 0.004162818659096956, -0.5198491811752319, -0.01684422977268696, -1.1659412384033203, -0.759528636932373, 1.9143998622894287, 1.2875993251800537, 0.12403065711259842, 0.5841834545135498, -1.3783724308013916, 0.3159996569156647, -0.3741213381290436, -0.9226585626602173, 0.24290569126605988, -0.6555785536766052, 0.9973818063735962, -0.45477306842803955, -0.9631419777870178, 1.32426118850708, -0.6965678930282593, 0.6272339820861816, 0.17410723865032196, -0.1602872908115387, -1.5210373401641846, 1.166191577911377, -0.9123797416687012, 0.0809539183974266, 2.004227876663208, 0.15407618880271912, 0.008350054733455181, 0.8121008276939392, -0.23110103607177734, -1.3781489133834839, 0.8497691750526428, -1.550412893295288, -0.2939148247241974, 0.34000730514526367, -0.3887307047843933, -0.30940842628479004, 0.8200430870056152, -1.7739427089691162, 0.46119800209999084, -0.9169033169746399, -0.4008443057537079, -0.79215008020401, -0.974338173866272, 1.0658305883407593, 0.9544181227684021, -0.39208394289016724, 2.373188018798828, 0.42025673389434814, 0.19765500724315643, 0.03575199842453003, 1.0404744148254395, -0.6643825769424438, -1.5470739603042603, 0.17340654134750366, -0.28193625807762146, -1.860477328300476, 0.007484983187168837, 0.9040926098823547, -0.8680633306503296, 0.09911425411701202, -1.308470606803894, 1.0427203178405762, -0.2556806206703186, 1.4108701944351196, -1.1794699430465698, 0.10463675856590271, 0.401540070772171, -0.7008029818534851], [0.37142759561538696, -1.402951717376709, 0.7086707949638367, -0.23358146846294403, -0.9794336557388306, -1.8911762237548828, -0.027329372242093086, -0.2971726357936859, -0.5520328283309937, -0.9090096950531006, 1.8395100831985474, 0.4737985134124756, 0.7070008516311646, -0.6059086918830872, -1.679520606994629, 0.8493273258209229, -0.5618044137954712, -0.5304853916168213, -1.3938418626785278, -1.6207764148712158, -2.014664649963379, -1.0852240324020386, 2.2010655403137207, -0.13446582853794098, 0.07975992560386658, -0.27175331115722656, -0.2587456703186035, 2.1969854831695557, -2.7330713272094727, -0.5170018672943115, -1.6152656078338623, -0.6191781163215637, -0.1801712065935135, 1.0807908773422241, 0.913072407245636, -1.616672158241272, 1.471084713935852, -2.2103614807128906, 1.3813509941101074, 1.644966959953308, 0.8721230626106262, 1.4651472568511963, -0.5170511603355408, 0.14949384331703186, -0.017072072252631187, 1.2287429571151733, -0.2744937837123871, 0.635829508304596, 0.9103612899780273, -0.33052971959114075, -0.6626389026641846, -0.4141973853111267, -0.1267838478088379, -0.40075668692588806, -1.5414891242980957, 0.6396909356117249, 0.17342044413089752, -0.6266544461250305, 0.34785833954811096, 0.6741167306900024, 0.6332198977470398, 0.09377934038639069, 1.746209979057312, -1.6784807443618774, -0.33659231662750244, -0.12077819555997849, -1.5765846967697144, -0.29140153527259827, -0.025003118440508842, -0.1271185427904129, -1.2803090810775757, 0.40311846137046814, -0.13720031082630157, -0.2601802945137024, -0.9644997715950012, -0.26037275791168213, 0.13730403780937195, -1.441907525062561, 1.7270560264587402, 0.03630264848470688, 0.5047523975372314, -0.6018571257591248, 0.7151466012001038, -0.5812039375305176, 0.8755007386207581, 0.49754709005355835, 0.724782407283783, 0.5818828344345093, -0.17856532335281372, 0.5634610652923584, 0.9617671370506287, 1.2912623882293701, 0.5379529595375061, 0.4238773286342621, 0.6832599639892578, -1.0873479843139648, -0.12958811223506927, 0.3922120928764343, -1.1514039039611816, -0.08084506541490555, -1.112207293510437, -0.9333618879318237, -0.7411324977874756, -0.660373866558075, 0.17824237048625946, 0.4298473596572876, -0.625575840473175, -0.5790404081344604, 0.384259968996048, 0.022533204406499863, 0.06769521534442902, 0.9329521059989929, 0.38429221510887146, 1.1478979587554932, 0.6175681948661804, -1.804099440574646, 0.18078981339931488, 0.06875511258840561, 0.2761569023132324, -1.4150331020355225, -1.3936967849731445, 2.248922824859619, 4.358604907989502, -0.23570095002651215, 1.5814933776855469, -0.823512613773346, 0.4220164716243744, 1.0350184440612793], [-0.047910597175359726, -0.1393030434846878, 0.054913874715566635, 1.758438229560852, 0.6383301019668579, -0.7584896683692932, 1.2260525226593018, -1.2734997272491455, 0.4472614824771881, 1.4028748273849487, -0.6464200615882874, 0.169411301612854, -0.9020417332649231, -0.1666932851076126, 0.2422104924917221, 0.3314616084098816, -1.0955970287322998, 1.2972856760025024, 0.009398405440151691, 1.51398503780365, -0.7801210880279541, 0.8113806247711182, -1.0747557878494263, 1.0939078330993652, 1.0788826942443848, 0.6662203669548035, 1.5732250213623047, -0.2813795506954193, 0.44873368740081787, 0.445791631937027, -0.6618408560752869, -0.33187735080718994, -0.26654890179634094, -0.13627228140830994, 0.8808578848838806, -0.3244016766548157, -0.21212051808834076, -0.6115747094154358, 1.4917651414871216, -0.7600032091140747, 0.19628283381462097, 1.0864136219024658, 0.10372629016637802, -0.3908657133579254, 0.5759220123291016, 1.2870007753372192, 1.3339016437530518, -0.19088804721832275, 1.738630771636963, 1.3258262872695923, -0.3083176612854004, -0.4976717531681061, -0.1827106475830078, 1.0558674335479736, 0.28500446677207947, 0.49531692266464233, -0.5154137015342712, -1.25505530834198, 1.5361937284469604, -0.7436500787734985, 0.731525182723999, -2.064697504043579, -0.10298340022563934, -1.2472832202911377, 1.3608779907226562, 2.212216377258301, 0.471732497215271, 0.45671606063842773, -1.4924805164337158, -0.4603644907474518, -0.2151189148426056, 0.8933348059654236, 1.7212263345718384, 1.2845731973648071, -0.15953011810779572, 1.0322861671447754, 1.4651908874511719, -1.0916228294372559, 0.7146594524383545, 0.38602590560913086, -0.4392760992050171, 0.4136350750923157, 0.057581834495067596, 0.017004765570163727, 0.18835189938545227, 0.19570426642894745, -1.4730565547943115, 2.614718437194824, 1.014366865158081, -0.2531515657901764, -1.433345913887024, 0.873544454574585, -1.6838988065719604, 0.030908750370144844, -0.7066220641136169, -0.025289997458457947, -1.0779407024383545, -0.7191352844238281, -0.6663898825645447, 0.3611823320388794, -1.1544352769851685, -0.7124518752098083, -0.9440277218818665, -0.10561036318540573, -0.7616608738899231, 1.1701279878616333, 1.3875001668930054, 0.5056098103523254, -0.784404456615448, 0.10508893430233002, -0.36758843064308167, -1.421043038368225, -1.2399823665618896, 0.1802535504102707, 0.9459788203239441, 0.9518548250198364, -0.8940567374229431, 1.17146897315979, -0.811478316783905, 0.3368326723575592, 0.5490419268608093, 0.9618152976036072, -1.9054031372070312, -0.6247465014457703, 0.22914084792137146, -0.486115425825119, -0.24334973096847534, 0.2850172817707062], [-0.0665978342294693, 0.5774744153022766, 1.6174507141113281, -0.2825172245502472, -1.8700740337371826, -0.6970918774604797, -0.395900696516037, 1.2494685649871826, -0.5729560852050781, -0.7319042682647705, 0.6554646492004395, 0.0033705884125083685, -0.7323703169822693, 0.1743689328432083, -0.6563730239868164, -0.4941580891609192, -1.422794222831726, 1.2093371152877808, 0.41433367133140564, 0.5367137789726257, 0.7904012799263, 0.9505255222320557, -0.42106911540031433, -0.943786084651947, -0.15301190316677094, 0.1850252002477646, 0.10606583952903748, -0.6224386096000671, 0.23419328033924103, -2.4404194355010986, -0.5793574452400208, -0.26594123244285583, 1.4924565553665161, -0.09462476521730423, -1.295303463935852, 1.1840102672576904, 0.014728960581123829, -0.5175852179527283, 0.1488344520330429, 1.054891586303711, 0.48088932037353516, -0.0030144795309752226, 0.9749372601509094, 0.7114549279212952, -0.6494886875152588, 2.379976749420166, -0.45074930787086487, 0.006954123266041279, 0.7897757291793823, -0.33524587750434875, -0.10209210962057114, -0.9120083451271057, -0.7821508646011353, -2.0767667293548584, -0.10480006784200668, -0.3692735731601715, -0.1962941735982895, -0.8569450974464417, -0.9660685658454895, 0.9328314065933228, -0.8161674737930298, -1.241485834121704, -0.3049122095108032, 1.780051827430725, 0.04242604970932007, -0.8727832436561584, -0.38022086024284363, 1.3960939645767212, 0.6599118709564209, -0.7910925149917603, 1.1601473093032837, -1.198646068572998, 0.8994495272636414, 0.9600647687911987, -0.8187552690505981, 0.03882750868797302, 1.1049256324768066, 0.292883962392807, -1.237494707107544, -0.7561646699905396, 1.7331902980804443, -1.6042519807815552, 0.33373022079467773, 0.03678373992443085, 0.003540425095707178, -0.02353716269135475, -0.011711055412888527, -1.3504198789596558, 0.37839046120643616, -1.950056791305542, -0.12032521516084671, 1.2486064434051514, -0.9795184135437012, -0.20787182450294495, 0.10507408529520035, 0.9175941944122314, 0.38297683000564575, 0.1979532688856125, 0.18396106362342834, -0.04107293114066124, 0.10722744464874268, 0.20192226767539978, -0.6168932318687439, 0.32057905197143555, 2.240744113922119, -0.14122121036052704, -2.2184550762176514, 0.4194377362728119, -0.6020340919494629, 1.8234143257141113, -0.1335594207048416, 0.30089595913887024, -1.3471194505691528, -1.7394864559173584, -1.7725125551223755, 0.17948825657367706, -0.8274834156036377, 0.44727393984794617, -0.7457470297813416, -0.014067397452890873, -0.5051948428153992, -0.16149958968162537, -0.015246212482452393, 0.5492268800735474, -0.4191163182258606, 1.40302312374115, 0.4081328213214874, 0.9852411150932312], [-0.15183159708976746, -0.16706188023090363, -0.36575403809547424, -0.813857913017273, -0.41762542724609375, 0.5077590346336365, -0.18336908519268036, 0.650534987449646, -0.9972010850906372, 0.5439095497131348, 2.321788787841797, 0.6608576774597168, -0.5477578043937683, 0.03174911066889763, 0.01787007786333561, -0.03283609449863434, -1.1909259557724, 0.37187063694000244, -0.705643892288208, -0.49023905396461487, -0.769232988357544, -0.9869892001152039, 0.045447323471307755, 0.8051121234893799, 0.9339103698730469, 2.2181034088134766, 0.7102706432342529, -0.2279803305864334, -0.8742005825042725, -0.04599789157509804, 0.9695356488227844, 0.12678943574428558, 0.9377791285514832, 1.0718727111816406, -1.5964562892913818, -0.4105127453804016, 0.45795977115631104, 0.38448837399482727, 1.005271315574646, 0.5542900562286377, -0.7653142809867859, 1.3978465795516968, 0.2495957911014557, -0.2749869227409363, -0.19364352524280548, -0.5802819728851318, 1.771265983581543, -1.0933603048324585, 0.5816828608512878, -0.16719897091388702, -1.4775919914245605, -0.8762372732162476, 0.47042134404182434, 0.008640236221253872, -0.04630802571773529, 0.15472103655338287, 1.32123601436615, 0.31997957825660706, 0.4487183392047882, -0.1076895147562027, 0.6448518633842468, 0.10805486142635345, -1.173850655555725, 1.34564208984375, -0.08554978668689728, -2.5752601623535156, -0.5050281286239624, 0.1047452762722969, 0.5564173460006714, 1.2255849838256836, 0.3391115665435791, 0.3916318118572235, -0.7461783289909363, -1.4037253856658936, 0.14740155637264252, -0.09113136678934097, -0.823282778263092, 0.9682642221450806, 0.010856913402676582, 0.4527416527271271, 0.2840088903903961, 1.9214202165603638, -0.5658465623855591, -0.021328147500753403, 0.37143880128860474, 1.1274229288101196, -0.86142897605896, 0.6759090423583984, 0.2264501303434372, 0.46319645643234253, -1.049562692642212, 0.8611374497413635, 1.2667226791381836, 2.0867457389831543, 0.268170565366745, 0.9731405973434448, 1.132347583770752, 0.82026606798172, 0.48806577920913696, -2.041788101196289, -0.3364666700363159, -1.3525559902191162, 0.6552087664604187, 0.556949257850647, -0.5957756638526917, 1.1564074754714966, 1.8935662508010864, -2.073348045349121, -0.8099920153617859, -2.1011226177215576, -0.6283133625984192, -2.3193020820617676, 1.7438063621520996, -1.9446296691894531, 0.7839383482933044, 0.9304218292236328, 2.535193681716919, -1.8397020101547241, 0.42669224739074707, -0.1471589207649231, 0.6127016544342041, -2.9699037075042725, -0.5833072066307068, -0.3305637836456299, -2.1238298416137695, 0.6725002527236938, -0.07859666645526886, -0.7361447811126709], [0.6794998049736023, 0.022165250033140182, 1.9983729124069214, -0.016270671039819717, 1.3445992469787598, -0.006795375607907772, 1.2963377237319946, 0.5573975443840027, 1.9467600584030151, 0.00995697919279337, -1.685526967048645, -0.1866704523563385, 1.0871446132659912, 1.0820629596710205, 1.083162784576416, -0.10938672721385956, 0.24177761375904083, 0.3949261009693146, 0.9519864320755005, 0.4348940849304199, 0.0918232724070549, 0.04351839795708656, -2.2091569900512695, -1.3321903944015503, -1.1192853450775146, 2.509307384490967, 0.016771186143159866, -0.03759114071726799, -0.08247633278369904, 0.08818496763706207, 0.6877713203430176, 1.4805632829666138, 1.1018741130828857, -0.9641140699386597, -1.0143227577209473, 0.5332155823707581, 1.9701824188232422, 0.3487509489059448, -0.8085014820098877, 0.3460134267807007, -0.13089419901371002, 0.6210657358169556, 0.6171274781227112, -0.27813011407852173, -1.0881807804107666, 1.4628416299819946, -0.3792543113231659, 0.5820035934448242, -1.1965789794921875, 0.7050354480743408, 0.06833545863628387, 0.4639655351638794, 1.226170539855957, 1.8802998065948486, 0.7803182005882263, 1.1224902868270874, 1.2576671838760376, 0.4358915388584137, -0.2005111277103424, -1.6030213832855225, 0.672620952129364, -0.09801211208105087, 0.7690466642379761, -0.4741082191467285, -1.390460729598999, -2.3296420574188232, 0.750062882900238, -1.9008721113204956, 0.15450012683868408, 1.2368346452713013, 0.043299317359924316, 1.161615252494812, 0.46120065450668335, -0.27856358885765076, -0.14605270326137543, 0.4435986578464508, -0.6659945845603943, 1.314245343208313, 0.09969472140073776, -2.2940900325775146, -0.3692382276058197, -0.08424046635627747, 0.553219735622406, 0.19113479554653168, -1.2296347618103027, -1.7395744323730469, -0.4984363615512848, -0.2217453122138977, 0.08748384565114975, -0.656165361404419, -0.3091678321361542, -1.219706654548645, 2.149620771408081, -0.6488037705421448, -0.7340041995048523, 0.18828709423542023, -0.8647592067718506, 0.7906521558761597, -0.811296284198761, 0.4728144407272339, 0.13225547969341278, 0.5449135303497314, 0.6481083631515503, -1.0212806463241577, -0.8784512281417847, 0.6519984006881714, -0.7346042394638062, -0.35908570885658264, -2.111783742904663, 0.6790763139724731, -1.4615757465362549, -0.4378073513507843, 0.35560137033462524, -0.5334811210632324, -0.02495720610022545, -1.5698981285095215, -0.013207743875682354, -0.6478542685508728, 0.16250157356262207, 0.41999316215515137, 1.6951730251312256, -0.5664528012275696, 2.274369478225708, 0.5554568767547607, -0.03947976604104042, -1.1551786661148071, 0.5683507919311523, 1.7537177801132202], [1.4277889728546143, -0.5077787041664124, 0.8002554178237915, -1.79495370388031, 1.5349373817443848, -0.7545810341835022, -0.8134079575538635, 1.4230668544769287, 1.5107998847961426, -1.2717067003250122, -1.0656572580337524, 0.8447075486183167, 0.07028065621852875, 1.2625774145126343, -1.4179013967514038, -1.7798969745635986, -1.2767452001571655, -0.4888570308685303, -0.8397715091705322, -1.6185520887374878, -1.0457857847213745, 1.134805679321289, 0.5267515182495117, 0.5681887865066528, 0.897834062576294, -0.8110030889511108, -0.22537213563919067, 1.214949369430542, 0.2802242636680603, 0.10825338959693909, -3.2027018070220947, -1.2408086061477661, -0.5212958455085754, -0.22428274154663086, -0.9261462092399597, -0.20188766717910767, 0.05240604653954506, -1.7948830127716064, 0.619334876537323, 0.5702977180480957, 1.6501483917236328, -2.069802761077881, 1.5997387170791626, 1.1321444511413574, -0.0061778961680829525, -0.9499396085739136, -0.7502478361129761, 0.9850296378135681, 0.7708854079246521, 0.13739249110221863, 1.733168363571167, -0.9347820281982422, 0.5757092237472534, -0.023166486993432045, 0.2689339816570282, -1.379711389541626, -0.07124168425798416, -0.8855209946632385, 0.7249020934104919, 0.21713747084140778, -1.01665461063385, -0.30404365062713623, 1.7901438474655151, 0.3239295482635498, 1.5807427167892456, 0.16414473950862885, 0.05537023767828941, -0.47874054312705994, -0.1309138834476471, 1.0794906616210938, -3.047710657119751, 1.851029634475708, -1.655150055885315, -1.6001734733581543, -0.9985303282737732, -0.1667707860469818, 1.4158899784088135, -1.1427414417266846, 1.0288389921188354, -0.3201514184474945, 1.565041184425354, -0.669044017791748, -0.29232436418533325, 0.274644136428833, 0.6657426953315735, 0.7816619873046875, -0.8288454413414001, -0.22105242311954498, -0.9952560663223267, 1.2569853067398071, 1.922616720199585, -0.19988591969013214, 0.0738605409860611, -1.0520528554916382, 1.5278041362762451, 1.4715322256088257, -0.42255571484565735, -0.5377863645553589, -0.18464595079421997, -0.7561538219451904, 0.4606516659259796, -0.1485259085893631, 0.5927557349205017, -0.3545779585838318, 1.4825801849365234, -0.16783633828163147, -0.359992116689682, 0.6829591393470764, -0.6223347187042236, -0.08851414173841476, 1.5245157480239868, 0.17096726596355438, 0.0858621746301651, 0.42434850335121155, 0.334597647190094, -0.37430456280708313, 1.5342234373092651, -0.7231512665748596, -0.8078476190567017, 1.4873846769332886, 0.3477611243724823, 0.7868952751159668, -0.2714673578739166, -0.048451684415340424, -1.9503469467163086, 0.6328138709068298, 0.6094080805778503, -1.758577585220337], [-0.8413662314414978, 0.14218489825725555, 0.09139638394117355, -0.4023362100124359, 0.27367743849754333, -0.9608399868011475, -0.33905228972435, -1.1466761827468872, 0.7390438914299011, -0.9431633949279785, 0.9655191898345947, 1.0279110670089722, -0.09924530237913132, 0.8274127840995789, -0.8696795701980591, 1.1360807418823242, -0.370668888092041, -1.2434724569320679, 0.04063122346997261, 1.7501180171966553, 0.44425204396247864, 0.5171027779579163, 0.10495748370885849, -1.3535064458847046, -0.6474796533584595, 1.3862189054489136, 1.4134467840194702, 0.3655407428741455, 1.157476782798767, -1.1865572929382324, -1.0171012878417969, -0.1753108948469162, -1.4897027015686035, -0.07828713208436966, 0.4671855866909027, 0.7588992714881897, 0.08910617232322693, 0.5883067846298218, -1.5189094543457031, -0.7937068343162537, -1.199654459953308, 1.309449553489685, 0.9991366863250732, 0.1678904891014099, 0.3677726089954376, -0.6340608596801758, -0.30520662665367126, 0.3435458242893219, 1.2312486171722412, -0.16396769881248474, 0.24657638370990753, -1.1412065029144287, -0.2388608753681183, 0.049774594604969025, 0.7950325608253479, -1.1128543615341187, 0.5820239782333374, 1.0237621068954468, -1.3649959564208984, 0.0840863585472107, -0.17730282247066498, -0.6381618976593018, 0.05722283199429512, -0.5898563861846924, 0.3671983778476715, 0.7055893540382385, 0.5365012288093567, -0.36559173464775085, -0.29200613498687744, 0.33765020966529846, -0.566688060760498, -0.3845818042755127, -0.9701922535896301, 0.05194608122110367, -1.0157654285430908, 0.8879066109657288, 0.8834234476089478, -0.45682263374328613, -0.8718293905258179, 0.09452907741069794, -0.28976115584373474, 0.5935054421424866, -0.8426036834716797, 0.32462525367736816, -0.3573553264141083, 0.5287622213363647, -0.5306700468063354, -0.6688032746315002, 0.4372193217277527, -1.289649248123169, 0.042190324515104294, 0.2621501386165619, -0.14636220037937164, -0.7251753211021423, 1.4819201231002808, -0.7888250350952148, -0.06557520478963852, -1.0719163417816162, 0.7068378329277039, -0.25597214698791504, -0.6675506830215454, -0.22071997821331024, -0.8154217600822449, -1.0417660474777222, 0.6458847522735596, -0.6191310882568359, 1.28413724899292, -0.5260056853294373, 0.3652370870113373, -1.108115315437317, 0.1411089450120926, -0.03738950192928314, -1.81374990940094, 0.9755957126617432, -0.13007943332195282, -0.4232534170150757, -0.1007009893655777, -0.2434919774532318, -0.09113409370183945, 0.9342483282089233, 1.0660958290100098, 1.3439266681671143, -0.6533997058868408, 0.4666486084461212, -0.024484509602189064, -0.29121142625808716, -0.1517532914876938, -0.5599823594093323], [-1.2374067306518555, 0.1319171041250229, 0.7481585741043091, 0.366456001996994, 0.5849223732948303, 0.35707908868789673, -1.0044788122177124, -0.1925763040781021, 1.60175621509552, -1.604398488998413, -0.8917941451072693, -1.3288952112197876, 0.7053176164627075, 0.0986725315451622, 0.3154184818267822, -1.3797446489334106, 0.6746628284454346, -1.1057026386260986, -0.08152831345796585, 0.11902347952127457, -0.7683528661727905, 1.6794012784957886, -0.06797205656766891, 1.3472681045532227, 0.03827684745192528, 1.0302636623382568, -0.8181673288345337, 1.4843844175338745, -0.5521268844604492, 0.2618122696876526, 1.3958796262741089, 0.5884624719619751, 0.6099175810813904, -0.06572728604078293, -2.1050267219543457, 0.98062664270401, -1.9217005968093872, 0.7752792835235596, -1.1923433542251587, 0.03226356580853462, -1.1799505949020386, 0.03117888607084751, 0.7462127804756165, 0.930776059627533, 0.624301552772522, 1.2160123586654663, -1.329197645187378, 0.214454784989357, -0.2977760136127472, 0.4112553894519806, 0.8893689513206482, -0.6401824355125427, -2.796243190765381, -1.2551714181900024, -1.5263313055038452, -0.17816969752311707, -0.8939415216445923, -0.8740542531013489, 0.3726027309894562, -0.4716539978981018, -0.5185637474060059, 1.0864803791046143, 0.496930330991745, 2.1910343170166016, 0.8954796195030212, 0.44169363379478455, -0.5645033717155457, -1.3263754844665527, -0.07012781500816345, 0.15996012091636658, -1.4239617586135864, 0.06409171968698502, 1.2471643686294556, -0.5211272835731506, -2.4480090141296387, 2.0029385089874268, -2.525946855545044, 0.542710542678833, 0.519729495048523, -1.8121840953826904, 0.22227932512760162, 0.47346264123916626, -0.08002375811338425, -0.037274740636348724, -0.6907032132148743, 0.9445055723190308, 0.286235511302948, -0.3308540880680084, 1.3979052305221558, -0.6096917986869812, -0.38148894906044006, 0.5375292301177979, -0.4597528874874115, -2.465935230255127, -0.49489110708236694, 1.390336275100708, 0.5878660082817078, -1.5343939065933228, 0.8751912713050842, -0.8100041747093201, -0.7356027364730835, -0.8278424739837646, 0.8982670307159424, 0.7954531311988831, -0.6614052653312683, -0.46670299768447876, 0.26910102367401123, -1.0850597620010376, -0.39695659279823303, -0.20707261562347412, -1.2028508186340332, 1.6049163341522217, -0.5739596486091614, 0.5852039456367493, 0.14400391280651093, 1.8218978643417358, -1.0872536897659302, -0.012571997940540314, -0.08832374960184097, 0.8405089974403381, 0.9367384314537048, -2.284715175628662, 1.181793451309204, 0.04248080402612686, -0.7200721502304077, -0.6790975332260132, 1.5437830686569214, 2.278428077697754], [1.806216835975647, -1.0932880640029907, 0.961868405342102, -0.12899833917617798, 0.6462434530258179, -0.5202961564064026, 1.9361563920974731, 0.8113605380058289, -3.276655673980713, 0.31526586413383484, -0.6711584329605103, 0.9026604294776917, -0.48827865719795227, 1.227539300918579, 0.08346289396286011, -0.5429620742797852, -0.8584861159324646, 0.3690301775932312, 0.06272808462381363, -0.516524076461792, -0.14042212069034576, 2.790799140930176, 0.4599955081939697, 0.5305315256118774, 2.396843194961548, -0.2933540940284729, -0.18719452619552612, 0.0524897538125515, 0.9205911159515381, 1.7140519618988037, -0.3593604266643524, 2.219280242919922, 0.5377029776573181, 0.06068594008684158, -0.9461219310760498, -0.16645191609859467, 1.5460656881332397, -1.7814687490463257, 0.1017579659819603, 0.631578803062439, -0.4345226585865021, 0.49427467584609985, -1.0574253797531128, -1.5021356344223022, 0.06608245521783829, -2.166550636291504, 0.6852281093597412, -1.1681674718856812, 0.20286938548088074, 0.32120686769485474, -0.6787460446357727, 0.2904095947742462, 0.9030964374542236, -0.0633198544383049, -0.7806010246276855, -0.29861536622047424, 0.65696781873703, -0.06953359395265579, 0.48154839873313904, 0.31266671419143677, -0.20981904864311218, 1.4463322162628174, 0.8566976189613342, 0.21929465234279633, 0.4808100759983063, 0.41595175862312317, -0.014233370311558247, -0.2700195610523224, 1.8429145812988281, 1.3579639196395874, 0.7533789277076721, -0.989749550819397, 0.7713146805763245, -1.414799451828003, -0.9398747682571411, 0.9406393766403198, -0.8852676153182983, 0.11399899423122406, -0.09712854772806168, -0.832804262638092, 0.6802981495857239, -0.34988728165626526, -1.969762921333313, -0.43291977047920227, 0.9061086177825928, 0.8838029503822327, -1.7460174560546875, 0.25922515988349915, 0.21992817521095276, -0.48194757103919983, -1.341658353805542, -0.2679000794887543, -0.0049408674240112305, -0.46736958622932434, 1.3587241172790527, -0.9510694742202759, -0.47052037715911865, 0.6762376427650452, 0.3326651453971863, 0.5379707217216492, -0.3600950837135315, -1.380323052406311, 1.1979923248291016, 0.3965604901313782, 0.2691832184791565, 0.9971886873245239, -0.18517716228961945, 2.219683885574341, 0.7436943650245667, -0.7378569841384888, 1.8532209396362305, -0.03980045020580292, 0.8079771399497986, -0.5243882536888123, 0.6331892013549805, 0.3362240791320801, -1.0797981023788452, -0.7458600401878357, 0.3155857026576996, 1.1419525146484375, 1.868233323097229, 1.7345176935195923, -1.0096919536590576, 0.431824654340744, 0.4929307699203491, 0.5979518890380859, -0.3116719126701355, -0.40348923206329346], [-0.6493980884552002, -1.2313076257705688, -0.15330950915813446, 0.151123046875, 0.028427893295884132, 0.10936063528060913, -0.16750335693359375, 0.7709605097770691, -3.7131946086883545, 1.7922718524932861, 1.0178897380828857, 0.008421370759606361, 1.6079857349395752, 0.13817648589611053, -0.6833368539810181, 0.087992824614048, 0.20451244711875916, -0.005334703717380762, 0.40824928879737854, -1.2435680627822876, -0.004096247721463442, 0.7777735590934753, 0.581239640712738, 0.33214402198791504, 1.039829969406128, 0.7874294519424438, 0.41334906220436096, 0.8416383862495422, 2.4396839141845703, 1.1085566282272339, -0.005371507722884417, -1.122847080230713, 0.2506505250930786, 1.117538571357727, 1.0013700723648071, -0.8800936341285706, -0.9210155606269836, -1.1063144207000732, -0.17341478168964386, -1.1279152631759644, 0.9006671905517578, 1.02829110622406, -1.8481101989746094, -0.9202346801757812, 1.582478404045105, 2.2932822704315186, 0.1572231650352478, 0.4116058349609375, 0.9056976437568665, 0.362693190574646, 0.9676489233970642, 0.7465802431106567, 0.5608088970184326, 1.4733223915100098, -0.31890904903411865, 0.38800013065338135, 1.2889324426651, -1.1102663278579712, 0.07087210565805435, -0.5198588967323303, -0.17685893177986145, -0.365958571434021, -0.007983208633959293, 0.06304612755775452, -1.2930783033370972, -1.8248964548110962, -0.9325624704360962, -0.5122173428535461, -0.4365275800228119, 0.12884213030338287, 0.4009254276752472, -1.1211344003677368, -0.37784233689308167, -0.0937456265091896, 0.7564108967781067, -0.8630189299583435, -1.1054210662841797, 0.0711621642112732, 0.11285143345594406, -1.2829797267913818, 1.5123558044433594, -0.09951838850975037, 1.3491706848144531, 0.38974806666374207, -0.17619763314723969, -1.0042123794555664, 0.023090828210115433, 1.1095598936080933, 0.5248405933380127, 0.5506985187530518, 1.1635178327560425, 0.4359854459762573, 0.869935929775238, 0.7290284037590027, -0.5458539128303528, -0.07736720144748688, -0.8864478468894958, 0.6564632058143616, -0.7342329025268555, -0.04358510300517082, -0.7333071827888489, -0.7949109077453613, -0.7310415506362915, 0.3270404040813446, -1.5783988237380981, -0.1245674416422844, 0.7444038987159729, -0.1220511794090271, 1.062104344367981, -2.578775405883789, -0.6604065299034119, -1.360554814338684, 1.3179036378860474, 0.4464256763458252, -2.128887891769409, 1.0999279022216797, 0.45799505710601807, 0.4924255907535553, -0.01666150614619255, 0.8360891938209534, 1.4305038452148438, -1.997174620628357, -0.7491733431816101, 0.19089196622371674, -2.8785741329193115, 0.0015800341498106718, -0.0406772643327713, -0.9702373147010803], [0.6293389797210693, -1.2323178052902222, 0.5518689751625061, -0.22086815536022186, 1.6049516201019287, 1.0997450351715088, 0.85740727186203, -0.387428879737854, -0.6133762001991272, 0.6680567860603333, 1.0095361471176147, 2.214853525161743, 2.4572198390960693, -1.4192063808441162, 0.6911953091621399, -0.20914001762866974, -1.8512654304504395, 1.1735568046569824, 0.07114937901496887, -1.5506680011749268, -1.7855993509292603, 0.38749420642852783, -1.9601633548736572, -1.3476855754852295, 1.54068922996521, 0.3368638753890991, -0.5582924485206604, -1.1014782190322876, 0.7437005043029785, 1.1730464696884155, 0.44392573833465576, 0.02380966767668724, -0.7440587878227234, -0.4994112551212311, -0.605566143989563, 0.16248859465122223, 0.7694739103317261, -0.7135207056999207, 1.3883084058761597, 0.2934929132461548, 0.22288395464420319, -0.24134433269500732, 0.23924775421619415, 0.14910773932933807, 0.5698134899139404, -0.09626442939043045, 0.7095438838005066, -0.37508460879325867, 0.30813634395599365, -1.1425281763076782, 1.6797009706497192, 0.5479072332382202, 1.151381254196167, -0.16586270928382874, -0.4230421185493469, 2.32999587059021, 0.6147499084472656, -0.3689141869544983, -0.5116653442382812, -0.14579558372497559, -0.37477588653564453, -1.076573133468628, 1.05698561668396, 1.3260338306427002, -0.08848357945680618, -0.14728526771068573, -0.5989762544631958, 0.8108398914337158, -0.919965386390686, -0.8573310971260071, -0.7034448385238647, -1.0827959775924683, -2.5786807537078857, -0.8355085253715515, 0.4268539845943451, 0.8419243693351746, -1.1450039148330688, -1.165050745010376, 0.7579718828201294, 0.35502251982688904, -2.3052306175231934, 0.09446683526039124, 1.3934288024902344, 1.7678728103637695, -1.3392229080200195, 0.198526531457901, -0.492017924785614, 0.41112831234931946, 0.4730294346809387, -0.13328894972801208, 0.8508749008178711, 1.0288559198379517, -0.8597478866577148, 0.8596762418746948, -0.3003794550895691, 0.02098204754292965, -0.5247394442558289, 0.461501806974411, 0.18564476072788239, -0.36018994450569153, 0.24457788467407227, -0.010805689729750156, 0.4978843629360199, 0.8985664248466492, 0.23204956948757172, -0.5540159344673157, -0.8946111798286438, 0.5525700449943542, 1.2507599592208862, -0.8484118580818176, 0.7266311645507812, 0.6594186425209045, 1.6321860551834106, -0.5728927850723267, -0.1947728842496872, 0.7648059725761414, -2.105863571166992, -0.9914659261703491, 1.1452217102050781, 0.815880537033081, 0.9885439276695251, -0.47867196798324585, -0.9585031867027283, 0.33665210008621216, -1.3517409563064575, 0.7078169584274292, 0.9507821798324585, 1.859220027923584], [0.23256883025169373, 0.14893107116222382, 0.4006834626197815, 0.5695340037345886, -1.1683515310287476, -0.5318135619163513, 1.0440446138381958, 1.1217008829116821, 0.48296311497688293, -1.083972692489624, -0.1219961941242218, 0.7954056859016418, 0.3097434639930725, 0.7094297409057617, 1.7039662599563599, 0.6909621953964233, -0.117024265229702, -1.0056171417236328, -0.05433839187026024, 0.1856183260679245, -0.14738363027572632, -0.11249155551195145, -0.3644011616706848, -1.6413136720657349, 0.07393023371696472, -0.6432235836982727, -0.1340857297182083, 0.4142701327800751, 0.5916518568992615, -0.23550698161125183, -1.819614291191101, 1.2903016805648804, -0.5850750803947449, 0.5028890371322632, -1.0784271955490112, 0.28704842925071716, 0.9181360602378845, 0.2087009996175766, 1.352484941482544, 0.3034522235393524, 0.4614005386829376, -0.40192699432373047, -1.262380599975586, 0.07470215857028961, 0.6408213973045349, -0.9683411121368408, -0.3337589502334595, 0.5936182141304016, -0.17669737339019775, 1.2933703660964966, -0.9539933800697327, 0.5168882608413696, 0.3772916793823242, -1.1843253374099731, -1.1729357242584229, -0.3374764621257782, 0.805872917175293, 0.11751450598239899, 1.6718101501464844, -0.7453253269195557, 0.8311418294906616, -0.5027514696121216, -0.18009856343269348, 0.5648121237754822, 1.9417487382888794, -0.9632051587104797, 2.346919536590576, 0.9173508882522583, -1.5860105752944946, -0.5814631581306458, 0.8185911774635315, 0.6439107656478882, -0.5142782330513, -0.7061656713485718, 0.2936881184577942, -1.8021138906478882, -0.4767981469631195, 0.5637183785438538, -0.8768244385719299, 0.438191682100296, 0.39097362756729126, -1.1031358242034912, -0.16645567119121552, -0.6234481334686279, 1.8551338911056519, -0.6643118262290955, -1.2350611686706543, -1.1180816888809204, -0.23148982226848602, 1.3917555809020996, -0.5157474279403687, -1.072917103767395, 0.8896019458770752, 0.7163299918174744, -1.0060707330703735, -0.6579412817955017, 0.6983143091201782, 0.3178803026676178, -2.2324600219726562, -1.9343633651733398, -0.9235250949859619, -1.4722726345062256, -0.4137290120124817, 0.2142656296491623, -0.7880166172981262, -1.1137346029281616, -0.21381817758083344, 0.514136791229248, -1.661800503730774, 0.854087233543396, -1.9512596130371094, -0.1570800244808197, 0.5129227042198181, -0.8518004417419434, 0.7358405590057373, -1.4720878601074219, 0.058233555406332016, 0.05361436679959297, 0.950714647769928, 0.700525164604187, -0.734490156173706, 0.2664984166622162, 1.1074422597885132, -1.6336740255355835, 0.620231568813324, -1.9318687915802002, -0.03283671662211418, 0.09924031794071198], [-1.205223560333252, 0.8746657371520996, 1.5008940696716309, 2.101803779602051, -0.2896512448787689, 0.5169114470481873, 1.4018222093582153, -0.3937035799026489, -0.2496112436056137, -0.5824032425880432, -0.34858596324920654, -0.5260960459709167, 2.2783477306365967, 0.9681012630462646, -0.8902022242546082, 0.9301971197128296, 1.3722702264785767, -0.2486930936574936, -0.08350320160388947, -2.4223008155822754, 0.037973977625370026, 3.057913064956665, -1.479621410369873, 1.2142537832260132, -0.04487494006752968, 0.6887165904045105, 1.41951322555542, -0.753973662853241, -0.8292925357818604, 0.7087421417236328, -0.20741631090641022, -0.98580402135849, 0.5993011593818665, 0.9748333692550659, -1.21880042552948, 0.5360379219055176, 0.2419537752866745, 0.5656833052635193, -1.164123296737671, 1.1679329872131348, -0.9094507098197937, 0.2710258662700653, 1.1627988815307617, -0.9072065353393555, -0.7428092956542969, -0.8323816657066345, 0.6908984184265137, -0.3799707293510437, -1.8784822225570679, -1.303341031074524, 0.36923977732658386, -1.3181177377700806, -0.6517296433448792, 1.3339786529541016, 0.37735655903816223, -0.19472044706344604, -0.5160084962844849, 0.10707200318574905, 0.520512044429779, 0.7965351939201355, 1.6921442747116089, 0.607329249382019, -1.387486219406128, 0.7472618818283081, 1.779399037361145, 1.1303001642227173, 1.7417670488357544, 0.7718486785888672, 0.5304502248764038, 1.184590458869934, -1.3180676698684692, 1.2084873914718628, -0.5782754421234131, 2.334851026535034, -0.6897886991500854, 0.5165376663208008, -0.7212166786193848, 0.5831676125526428, -2.227019786834717, -1.6831101179122925, 0.6411852836608887, 0.41820892691612244, -0.674322247505188, 0.777671754360199, 1.0515695810317993, 1.165879487991333, 0.3497837781906128, 1.0466750860214233, 1.7201615571975708, 0.5239570736885071, -0.9576976895332336, 0.7498863339424133, 0.4463096559047699, -0.01943812519311905, -0.889672577381134, 0.13989152014255524, 0.15881648659706116, 1.0971689224243164, -0.5455542206764221, 0.6562996506690979, 0.7773661613464355, -1.496648907661438, 0.5211547017097473, 1.131865382194519, 0.36632266640663147, 0.5368903875350952, -0.1482384204864502, -0.14270146191120148, -1.7077014446258545, -0.7957972884178162, -0.056104470044374466, -1.7291065454483032, -0.6712055802345276, 1.2710456848144531, 0.6650951504707336, 1.3448399305343628, 0.19329340755939484, -0.5923711061477661, -0.04186253622174263, 0.07060450315475464, -1.5282355546951294, 0.2361527681350708, 0.25325384736061096, 1.9340097904205322, -0.6967166662216187, 0.23156622052192688, -0.14201056957244873, -2.155380964279175], [0.6336867809295654, -1.0577912330627441, 0.9074109792709351, 0.1246655136346817, -0.546283483505249, -0.10203981399536133, -1.1892659664154053, -1.665223479270935, 0.5280290246009827, 1.8045001029968262, -1.063030481338501, -0.4841597080230713, 0.35007986426353455, 0.6555548310279846, 0.11152864992618561, -0.2629883289337158, -1.3654001951217651, -0.3400450646877289, 0.5938024520874023, 0.08000099658966064, -1.0559091567993164, -0.4623412489891052, -1.124861240386963, 1.3852458000183105, -0.4234565794467926, 0.26778092980384827, 0.3423154354095459, -0.30313965678215027, -1.6381367444992065, -1.224663257598877, -0.07702656090259552, 0.5898023247718811, 2.1750316619873047, -0.592269778251648, 0.3801926374435425, 1.3039379119873047, -0.5029282569885254, -0.21798329055309296, -2.774738311767578, 0.4083634316921234, 0.2102031409740448, -1.6249151229858398, 1.856244683265686, 0.4564313292503357, 0.38780510425567627, -1.4712278842926025, 0.6836977601051331, 0.7574259638786316, -0.46964412927627563, 1.3390802145004272, -0.8165373802185059, -1.6418269872665405, 1.9136805534362793, -1.4225261211395264, 0.38802769780158997, 0.9256726503372192, -1.1087546348571777, -0.18859921395778656, -0.5858135223388672, -1.1190028190612793, -1.3310554027557373, 0.6213472485542297, 0.569987416267395, -0.13699328899383545, 0.202327698469162, -0.3524637520313263, -1.3761920928955078, -0.43787768483161926, -0.14581093192100525, -0.8291597366333008, 0.9368659257888794, 1.046487808227539, 1.3236539363861084, -2.5216596126556396, -0.788587212562561, 0.05915740877389908, -0.5502533912658691, 1.824096441268921, -2.174450635910034, -1.9041059017181396, 0.6246855854988098, 1.253431797027588, 0.6510488986968994, 0.0028234212659299374, -0.8314666748046875, 0.8249068260192871, 1.973860740661621, -0.13490356504917145, -0.01239765901118517, 0.024417953565716743, -1.7525416612625122, 1.5548841953277588, 0.6210021376609802, -2.0300114154815674, -0.41133350133895874, 1.4248450994491577, 1.43108332157135, -0.9268811345100403, -0.1244385689496994, -0.5484350323677063, -1.2814035415649414, -0.7068465948104858, 0.17998063564300537, -0.2659939229488373, -0.8350322246551514, -1.2244744300842285, 1.3132014274597168, 0.580829381942749, 1.6541080474853516, -0.897644579410553, -1.8574998378753662, 0.515155017375946, -0.013003059662878513, 0.3821653723716736, -0.030793646350502968, 0.2852318584918976, 1.6959413290023804, -1.4085383415222168, 0.4368542730808258, -0.49061885476112366, -0.8533781170845032, -0.2776786983013153, 1.093624472618103, -0.0775456354022026, 1.6176011562347412, 2.4766616821289062, 0.6966267228126526, 0.19045038521289825]], [[0.4221762716770172, -1.902191400527954, 0.008282553404569626, 0.4705772399902344, 0.8560914993286133, -0.7167289853096008, -1.1855098009109497, 0.15081483125686646, -1.1438618898391724, 0.5024775862693787, 1.081787347793579, -0.30436450242996216, 0.5399956107139587, -0.9660223722457886, -0.7762117981910706, 0.5582449436187744, -0.5089054703712463, 1.5226809978485107, -0.4044267237186432, 0.9305208921432495, -0.584071695804596, -1.8174129724502563, 0.09741353243589401, -1.4300713539123535, 1.0606682300567627, 0.3671848475933075, 0.19808445870876312, -0.8695211410522461, 0.5497322082519531, -0.47768890857696533, 0.15361256897449493, -0.33148232102394104, -0.9141872525215149, 0.1472724825143814, 0.47437793016433716, 0.9202544093132019, 0.4461686313152313, 1.424659252166748, 0.6348012685775757, 0.8923318386077881, -1.3753769397735596, -0.31161925196647644, -0.1191888302564621, -1.7865314483642578, -0.09943690150976181, -0.2458808869123459, -1.870108723640442, 3.0478594303131104, -0.685525119304657, 1.7447028160095215, -1.5188795328140259, -2.3506572246551514, -1.365939974784851, 0.14008799195289612, -0.45286375284194946, -0.16762122511863708, -0.6129019260406494, 1.9694459438323975, -0.9254619479179382, 0.6574583053588867, -0.22709082067012787, 0.6681574583053589, -0.5936778783798218, -0.2944989800453186, 0.16512326896190643, -0.43744590878486633, -1.5745896100997925, 1.7187286615371704, 0.7288967370986938, 1.8629549741744995, -0.30572250485420227, -0.5929821729660034, 0.40102872252464294, 0.6098388433456421, 0.3446595072746277, -0.7619084715843201, -1.2314106225967407, 1.5100915431976318, 0.685460090637207, -1.04118013381958, 0.4459800124168396, -0.29555121064186096, -0.08423134684562683, 1.682490587234497, 1.8006367683410645, 1.0394096374511719, 0.07686279714107513, 1.2036986351013184, -1.0430139303207397, -0.5894535183906555, 0.03950236737728119, 0.2670247554779053, 1.7908111810684204, -1.1704379320144653, 2.318713426589966, -1.5740118026733398, -0.33998027443885803, -0.6614280939102173, 0.878229558467865, -0.3221193253993988, 0.4034338891506195, -0.4185650050640106, 0.724446177482605, -0.6155913472175598, 2.2658944129943848, 0.5225642919540405, -1.483944058418274, -1.526317834854126, 0.0016275899251922965, 0.7585504651069641, 0.13886426389217377, -0.1633293181657791, -0.15417109429836273, 1.1759272813796997, -1.1738107204437256, 0.8596953749656677, 0.4017733931541443, 1.0465854406356812, -0.6173635125160217, -1.1608294248580933, -1.3182920217514038, 0.02427266165614128, -0.8727105855941772, 0.6642785668373108, -0.28037333488464355, -0.8588337302207947, -0.7823120951652527, -0.2058178186416626], [-0.05424146726727486, 0.872286856174469, -0.08562353998422623, 0.48564568161964417, -0.5606606006622314, -0.7594602704048157, -1.4055854082107544, 0.12561939656734467, -0.8294714689254761, -1.0941599607467651, 1.689682960510254, -2.2576169967651367, 0.3372431695461273, -0.8863610625267029, -1.1887089014053345, -1.0896178483963013, 0.6310280561447144, 1.5237444639205933, -1.1830741167068481, 0.292621374130249, 0.8640609383583069, -0.3201601207256317, -0.30769145488739014, 0.4515177309513092, 1.4554002285003662, -0.17230810225009918, -0.522550642490387, -0.03644354268908501, 0.2593102753162384, 1.090872049331665, -1.5843307971954346, -0.18973682820796967, 0.10145167261362076, -0.8603060245513916, -1.6374070644378662, -0.517541766166687, -0.7570013999938965, -1.539420247077942, 1.2598254680633545, -1.331485629081726, 2.1998775005340576, -0.03640050068497658, -0.7654842138290405, 1.4892092943191528, 0.9119027256965637, 1.5021467208862305, -1.502855896949768, 0.8892492055892944, -0.29492029547691345, -0.8373346328735352, -0.8383605480194092, 0.43704724311828613, 0.1957998126745224, -0.7190788388252258, 0.6195268034934998, 1.1640547513961792, -0.5586012601852417, -0.9311804175376892, -0.1537790596485138, -0.6631792783737183, -1.624772310256958, -0.8345755338668823, 0.4111299514770508, 1.4989190101623535, -1.0367355346679688, 1.1932183504104614, -1.057474136352539, 1.336609959602356, 0.3889920115470886, 1.366084098815918, -0.878541111946106, 0.38110923767089844, 0.8501471877098083, 0.0742873027920723, 0.2901206314563751, 0.03710033744573593, 0.6299028992652893, 1.6355364322662354, -0.49086517095565796, -0.529926061630249, -0.5219476819038391, -0.10446150600910187, 0.3515474200248718, 1.3248939514160156, 1.1974960565567017, -1.728179931640625, -0.09082198888063431, 0.3748934268951416, 1.8732120990753174, 1.2389651536941528, -0.12871387600898743, 1.3506017923355103, -0.8853681087493896, 1.812290072441101, -1.215980052947998, 1.357801914215088, -1.7200899124145508, -0.39436373114585876, 0.18703220784664154, 0.9210429787635803, -1.3691518306732178, -1.018376350402832, 2.4041104316711426, -0.36739957332611084, -0.6085970401763916, 0.5876813530921936, -0.5292229652404785, 1.1322659254074097, 0.16949467360973358, 0.2759939134120941, 1.2108047008514404, 2.1701812744140625, -0.29628998041152954, -0.8231747150421143, -2.176877021789551, 1.855320930480957, 1.9865306615829468, 0.20121142268180847, -1.406350016593933, -0.676316499710083, 0.9546366333961487, 0.619215190410614, 1.0490844249725342, 0.2656985819339752, -1.5889675617218018, 0.5537973046302795, -0.49981653690338135, -1.0324256420135498], [-1.108912706375122, -0.7540464997291565, -0.9031195640563965, 0.6116282939910889, 0.06523700058460236, 2.3793599605560303, -0.7859820127487183, 1.138572335243225, -0.8723323941230774, -0.4182184338569641, -0.37107768654823303, 1.0787757635116577, -1.91645085811615, 0.7736122012138367, -0.349924236536026, -2.035118341445923, 1.0143859386444092, 0.9017177820205688, 0.7688358426094055, 0.0937306135892868, 1.9025968313217163, -0.850212037563324, -0.23350708186626434, -1.1343070268630981, -0.428475022315979, -0.8885717391967773, 0.18491701781749725, 0.07814900577068329, 0.0003854625392705202, -0.14785148203372955, 1.359099268913269, -0.41978758573532104, -0.7167510986328125, 1.309215784072876, -0.026794277131557465, 0.709090530872345, -0.8339105844497681, -0.666328489780426, 0.3434251844882965, 0.3491963744163513, -0.8755552768707275, 1.467819094657898, -0.5237184166908264, 2.825711727142334, -0.5746887922286987, -0.3472161591053009, -0.010229726321995258, -0.9262282848358154, -0.933463454246521, -0.4188646972179413, -1.1929627656936646, 0.14213542640209198, 0.5929714441299438, -1.4339526891708374, -0.03583306819200516, 1.0436152219772339, 1.7843695878982544, 0.8520148396492004, -2.5887844562530518, 1.3907649517059326, 1.2526811361312866, -0.5847566723823547, -0.5368769764900208, -1.0574859380722046, -0.008538434281945229, -0.2213711440563202, 0.9115810990333557, -0.5287327766418457, -0.43801549077033997, 0.13654084503650665, 0.7892128825187683, 1.226204514503479, -0.9045963287353516, 0.10522028803825378, 0.23452651500701904, -1.4979008436203003, 0.8668748140335083, -1.8153419494628906, 0.026453793048858643, 0.3440747857093811, -0.12079603224992752, 0.8429654240608215, -2.3771536350250244, 0.3199808895587921, -0.8087038993835449, 2.7646725177764893, -0.3649066090583801, 1.2744200229644775, -1.1123039722442627, 1.6099101305007935, -0.5000370740890503, -0.7388532757759094, -0.5423167943954468, 0.01539089810103178, -1.6357399225234985, -0.11275409907102585, 0.4838298559188843, 1.2300437688827515, -1.4897043704986572, -0.382132351398468, -0.7574489116668701, -2.1214795112609863, -1.3576014041900635, -0.42488595843315125, -0.054225705564022064, 1.016448736190796, -0.10627129673957825, -0.235836923122406, -0.046761106699705124, 0.18972021341323853, -1.1159964799880981, 1.2015914916992188, 0.5775015950202942, -0.04009591415524483, -0.26441651582717896, -0.8030701875686646, 0.7205603718757629, -0.5114012956619263, 0.6548307538032532, -0.1419210433959961, 0.09003978222608566, 0.09585785865783691, 0.06599199026823044, 0.5567724704742432, -0.3026197850704193, -1.9102140665054321, -1.4687572717666626, 0.6871206760406494], [0.1578742265701294, -1.4329135417938232, 0.6636931300163269, -1.4140520095825195, -0.8098229765892029, 0.9124383330345154, 0.21966998279094696, -1.5552440881729126, -0.35690540075302124, -0.11676575988531113, 0.995521068572998, 0.2592597007751465, 2.3631057739257812, -0.23951226472854614, -1.4783893823623657, 1.0542864799499512, 2.450101137161255, 1.9286211729049683, 0.7568379044532776, -0.5073022246360779, 0.3457850217819214, -0.3675524890422821, -0.14002983272075653, -0.2783631384372711, 0.237657830119133, -0.8222070336341858, 0.3827468156814575, -0.45310965180397034, -0.32094454765319824, 0.4936988055706024, -0.6492089629173279, 2.6109225749969482, -0.25608739256858826, -0.959718644618988, -0.29803112149238586, 0.8937270045280457, -0.19684173166751862, -0.2698657810688019, 1.1056121587753296, -0.9736944437026978, -0.33743998408317566, -2.7180371284484863, 1.2217543125152588, -0.34812989830970764, -1.2825710773468018, 0.48082178831100464, 1.4809151887893677, 0.8549278378486633, -1.5741279125213623, 0.9289118647575378, 0.7653252482414246, 0.11962349712848663, -0.9666866064071655, -1.3461205959320068, -1.1031526327133179, -0.07229525595903397, 0.7262735962867737, 0.43269068002700806, -1.4035292863845825, 1.2662978172302246, 1.2536516189575195, -0.5059496760368347, 1.143021583557129, -1.0444852113723755, -0.7480173707008362, -0.9851979613304138, 1.3579241037368774, 1.453370451927185, 0.07414095848798752, 1.85995614528656, 0.3316993713378906, 0.49117961525917053, 1.7294753789901733, 1.348141074180603, -1.4617953300476074, -0.6411370038986206, -0.8065968155860901, 0.373491108417511, 0.029470425099134445, 1.4152907133102417, 1.34416663646698, -0.25279513001441956, 0.42823365330696106, 0.660502016544342, 1.3539063930511475, 1.4744725227355957, -0.8341720700263977, -1.7507892847061157, -1.4533854722976685, 0.6976904273033142, 0.263118714094162, 0.6594910025596619, 1.8541516065597534, -0.46688610315322876, 0.9763702154159546, -0.9343783259391785, -1.0975338220596313, 0.6292855739593506, 1.8011929988861084, -0.673908531665802, -1.0573182106018066, 0.5324772000312805, 1.2265642881393433, -0.39475953578948975, -1.9943875074386597, 0.13695646822452545, 1.5529927015304565, -0.7220282554626465, -0.2976638376712799, 0.9983996748924255, 1.9554051160812378, -1.7620984315872192, -0.26058435440063477, -0.3402058482170105, 1.55862295627594, 0.43152812123298645, 0.38780102133750916, -1.1073262691497803, -1.4387284517288208, 0.131817027926445, -0.7817268967628479, 1.248476505279541, -0.022225020453333855, -0.06958746165037155, -1.3618844747543335, 1.5710426568984985, -1.7890896797180176, 0.25384166836738586], [-0.8290260434150696, 0.12844516336917877, 0.8366518020629883, 0.7863969206809998, 0.2289809137582779, -0.6090824604034424, 1.1019775867462158, 0.11082908511161804, -0.20544910430908203, -0.892261266708374, -0.6248691082000732, 0.12610627710819244, 1.6247671842575073, -1.4177861213684082, 0.5274792313575745, 2.524121046066284, 2.0519022941589355, -0.2969256043434143, 1.2190945148468018, -0.45406585931777954, -0.014878377318382263, -1.422423243522644, -0.523128092288971, -0.8993514180183411, 0.08937974274158478, -1.0271016359329224, -0.21167072653770447, 0.3195144534111023, -0.5149033665657043, 1.0116873979568481, -0.8121999502182007, -1.7477972507476807, -0.4064798057079315, -0.5558630228042603, 0.4803449511528015, -0.3066366910934448, -0.33043158054351807, -1.8518824577331543, -0.706524133682251, -1.1861045360565186, 1.8613039255142212, -1.650184154510498, -1.224182367324829, -0.7294656038284302, -1.4567692279815674, -0.33886614441871643, 0.5869255065917969, -0.6911224722862244, -0.9434921741485596, -0.39449286460876465, 1.3617546558380127, 0.48108556866645813, -1.3106043338775635, -0.6989078521728516, 0.016206517815589905, 0.10235048830509186, 0.7057076692581177, -1.1412662267684937, 1.111974835395813, 1.4504696130752563, -0.4276772141456604, -0.5140773057937622, -1.9091063737869263, -0.5346439480781555, -1.5523236989974976, -0.9725245833396912, -0.07564911991357803, 1.7441130876541138, -0.9844459891319275, -0.3731114864349365, 0.28249311447143555, -0.639868974685669, -0.14302201569080353, -0.8679107427597046, -1.1337863206863403, 0.612981915473938, 1.1643961668014526, -1.0319582223892212, -1.7004406452178955, -0.4568108320236206, 1.9851752519607544, 0.40042170882225037, -0.01740391179919243, -0.43825626373291016, -1.2093766927719116, 0.39790990948677063, 0.000893867458216846, -0.6710219383239746, 1.1439095735549927, 0.07380785793066025, 0.8093984723091125, 0.14155665040016174, -0.9467405676841736, -0.20102299749851227, 0.6759185791015625, -1.7435539960861206, -0.41333815455436707, -0.029099419713020325, 0.7792181372642517, -0.4821540415287018, 0.5150963068008423, -0.42833587527275085, 1.1476879119873047, 0.16481907665729523, -0.9733242392539978, 0.7774693369865417, 0.6770236492156982, 1.7289552688598633, 2.211055040359497, -0.3197300136089325, -0.19459430873394012, -1.1636641025543213, -0.6978513598442078, 0.011804603971540928, -1.059509515762329, -0.4091612994670868, 0.09119746088981628, -1.2713401317596436, -1.2558306455612183, 1.2314594984054565, 1.1786671876907349, -0.8076823949813843, -0.3756267726421356, -0.351818323135376, 0.07299210876226425, 0.6783424615859985, 0.2679399847984314, 1.2506183385849], [0.07745006680488586, -0.14640918374061584, -1.4294668436050415, 1.0986418724060059, 0.5041402578353882, -1.008851408958435, 1.0423882007598877, -1.341816782951355, -1.3275915384292603, 0.9266972541809082, -0.9519176483154297, 1.0394655466079712, -0.29983821511268616, -2.259002685546875, -0.4050203561782837, -1.3652927875518799, 1.1543192863464355, -0.5781174898147583, 0.47778233885765076, 1.0563477277755737, 0.6630719304084778, 0.5267364382743835, -0.7740660905838013, -0.3940005600452423, 0.10223057121038437, 1.407891869544983, 0.8046405911445618, -0.7414498329162598, -1.4129616022109985, 0.14574989676475525, -1.7378250360488892, -0.10471247881650925, -0.5943676233291626, -1.1012965440750122, 1.033559799194336, 0.5386255383491516, 0.6642988324165344, 0.10964015126228333, -0.8825318813323975, 0.5665019154548645, 0.7171658277511597, -1.5642473697662354, -0.41071003675460815, -0.6140273213386536, 0.1568707674741745, 0.5798662900924683, 0.19944022595882416, -1.5600478649139404, 0.6835896372795105, -1.6857151985168457, 1.516134262084961, 0.5424566864967346, -0.35070890188217163, -0.9850889444351196, 1.1447407007217407, 1.2809115648269653, 0.01945078745484352, 2.3412230014801025, -0.14387042820453644, 1.2811088562011719, -1.0022692680358887, -0.277157723903656, -0.3179335594177246, 0.6604352593421936, -1.829311728477478, -0.2227766364812851, 0.45029258728027344, -0.5362669825553894, 0.49623456597328186, -0.5374085307121277, 0.6141835451126099, -0.8284467458724976, -1.1760308742523193, 0.9167957305908203, 0.8162338733673096, -0.8536314368247986, -1.3717516660690308, -1.683058738708496, 0.7781111001968384, -1.0789676904678345, -0.42320579290390015, -0.16294236481189728, -0.616656482219696, 0.2238137125968933, -0.8269433975219727, 0.8604671359062195, -1.5569740533828735, 1.3208339214324951, 1.3410207033157349, -1.708586573600769, 0.4987187683582306, -0.9505448341369629, 0.3568190634250641, -1.9463635683059692, 0.18108247220516205, -0.4155963063240051, 1.3181114196777344, 0.5854516625404358, -1.6035221815109253, 1.0222923755645752, 0.3040412366390228, 0.622750997543335, 0.9810038805007935, 0.049204614013433456, 1.6816400289535522, -0.38383784890174866, 0.8379514813423157, -0.6327362060546875, 0.20297284424304962, 0.6147321462631226, -1.2311471700668335, -0.6161584854125977, -0.9163548350334167, 0.4006446897983551, -0.6759672164916992, 0.23064318299293518, -0.2490874081850052, 1.3750076293945312, -0.6342501044273376, -0.9286112189292908, -1.1338456869125366, 0.8793488144874573, 0.5782949328422546, 0.0927436575293541, 0.2128051221370697, 1.0684527158737183, 0.696195125579834, -1.1267118453979492], [-1.502143144607544, -0.35140910744667053, -1.604169249534607, 0.27957209944725037, -0.7096455097198486, 0.6022806167602539, 0.7823404669761658, -0.30190736055374146, 0.8243682980537415, -0.20376136898994446, 0.8595438003540039, -0.04889305680990219, 0.27797070145606995, 0.6525067687034607, 1.4253090620040894, -0.6727045774459839, -1.0349198579788208, 1.362988829612732, 0.8027312159538269, 2.0076091289520264, 1.5362628698349, -1.8981256484985352, 0.9364885687828064, -1.0611422061920166, -0.8970124125480652, -1.108862042427063, -0.902489185333252, -0.15767519176006317, 0.25353750586509705, 2.5154976844787598, 0.9050919413566589, -0.38045772910118103, -0.5959970951080322, -0.3456379175186157, 1.760640025138855, -1.1559171676635742, -1.3671897649765015, -1.5811415910720825, -0.8475883603096008, -2.9654104709625244, -0.8805544972419739, -0.20424148440361023, -0.38152945041656494, -1.0911133289337158, 0.7346197366714478, -0.7061954140663147, 0.8603448867797852, -1.3238641023635864, 2.0382096767425537, -0.026114709675312042, 0.5315886735916138, -0.04768000543117523, 0.24749703705310822, -0.5706679821014404, -0.7116336822509766, -1.1345213651657104, -1.9319846630096436, -1.2500261068344116, 0.33330753445625305, -1.5458338260650635, 1.5846318006515503, -0.19890138506889343, 0.5858151912689209, -1.2689687013626099, -0.8555582165718079, 0.8298286199569702, -0.8356248736381531, -0.17441000044345856, -0.6032894253730774, 1.3479317426681519, 0.6658378839492798, 0.5245952010154724, -0.858286440372467, -1.1974074840545654, 1.8619532585144043, 1.1577818393707275, 0.5590404868125916, -0.30013906955718994, -0.5836361050605774, -1.2343542575836182, -0.8782348036766052, 0.6466323733329773, 1.0950731039047241, -1.039537787437439, -0.37331560254096985, -0.11071471124887466, -2.3602547645568848, 0.6525458097457886, 0.6187396049499512, 1.0830631256103516, 0.44123613834381104, 2.053762435913086, -0.12888160347938538, -0.7581421136856079, -0.10554007440805435, -0.12190275639295578, 0.18310904502868652, -0.07096631079912186, 0.11944781243801117, -0.02618805505335331, -1.3747940063476562, -0.07264401763677597, 0.3350914716720581, 1.7179323434829712, 0.9243925213813782, -0.9401482939720154, 0.3986009657382965, -1.1940973997116089, -1.468471884727478, 0.7847919464111328, 0.6149274110794067, 0.5226753354072571, -2.415534019470215, -1.3144453763961792, -1.9667541980743408, 0.03532791882753372, -1.0216586589813232, -0.9753970503807068, -0.47371721267700195, -0.11134672909975052, -0.9271367192268372, -1.038382887840271, 1.2288143634796143, 0.1727726310491562, -0.7993922233581543, -0.5591652989387512, 0.6686611175537109, -0.6819080114364624], [-0.29404476284980774, 0.16825272142887115, -0.9356546998023987, -1.2763372659683228, 1.189976692199707, 0.7650464177131653, 0.7146546244621277, 0.3684329688549042, -0.4762856066226959, -1.6238573789596558, -0.40702420473098755, -0.31264081597328186, 1.6898821592330933, 0.7277380228042603, 0.18200597167015076, 1.4165244102478027, -2.258932590484619, -0.8221728205680847, -0.3113904297351837, -1.139298439025879, -0.6770504713058472, -0.49060890078544617, 0.3056933879852295, -0.4518781304359436, 0.5174608826637268, -0.9657156467437744, 2.3753347396850586, -2.353113889694214, -0.19707192480564117, 0.25210607051849365, 0.5861923098564148, -1.7759524583816528, 0.564805269241333, 1.015438199043274, -0.9574828743934631, -1.2527297735214233, -0.5963018536567688, -1.0489474534988403, 2.1371965408325195, -0.7137797474861145, 0.8272383809089661, -0.9308401346206665, 0.16979439556598663, -1.215684175491333, -2.0558817386627197, -0.3054538369178772, 0.5076383948326111, 0.647989809513092, 0.5578336715698242, -0.16245312988758087, -0.8975268006324768, -0.8992414474487305, -1.7245734930038452, 0.13943618535995483, 0.32190465927124023, 0.5805510878562927, 0.4804435074329376, 0.23251543939113617, 0.5547329783439636, 0.9705073833465576, -0.6043509840965271, 1.4066965579986572, -0.492521733045578, 0.5949237942695618, 1.297664999961853, 0.6795828938484192, -0.8444604277610779, 0.10250164568424225, 1.0401450395584106, 0.9053813815116882, 0.21764378249645233, -2.548384189605713, 0.5222189426422119, 0.5474036931991577, 0.07814513891935349, -0.9231178760528564, 1.5109566450119019, -0.8800934553146362, 0.28389039635658264, 0.49429455399513245, -0.22267921268939972, 0.34983834624290466, -0.44384196400642395, -0.20391681790351868, 0.4930048882961273, 0.9596779942512512, -1.0017553567886353, -1.1921745538711548, -1.2985856533050537, 0.3258766829967499, 0.516886830329895, 0.17530320584774017, -0.044525083154439926, 0.6364907026290894, -0.3432517647743225, 1.0183372497558594, 2.008061408996582, 0.34504204988479614, 1.8193635940551758, -0.5254372358322144, -0.7928223609924316, -0.45280569791793823, 0.2666614353656769, 0.9299958944320679, 0.7433581948280334, 0.8344129323959351, -0.00046405423199757934, -0.041577063500881195, -0.9399611949920654, 1.2155381441116333, -0.45553863048553467, 1.1036275625228882, 0.945512056350708, 0.7271876335144043, 1.3169509172439575, 0.8934482932090759, -0.01105226669460535, -0.6512966156005859, 2.107172966003418, -0.2088528424501419, -1.1172209978103638, -0.4257989525794983, 1.60347580909729, -1.1587722301483154, -0.10391324013471603, 1.1688798666000366, 0.31104153394699097, -2.907681465148926], [0.4952540993690491, -0.8229900598526001, 0.8481098413467407, -0.16068805754184723, 0.7750614285469055, 1.2288644313812256, -1.889620304107666, -0.20500805974006653, -1.4741332530975342, 1.9668444395065308, -1.6794906854629517, -0.6236470341682434, -1.3985902070999146, 0.2251153439283371, -0.05894654244184494, -0.4403003454208374, -2.2209415435791016, 0.6826305389404297, 0.6803534030914307, 0.5204090476036072, -0.32203209400177, 0.24822671711444855, -0.721235454082489, 1.2537204027175903, 0.12407614290714264, -1.3656890392303467, 1.4161436557769775, 0.3019334375858307, -2.538654327392578, -0.6610612869262695, -0.4506371319293976, -0.8008161187171936, -0.8114335536956787, -0.15961064398288727, -0.48095208406448364, 0.3674888610839844, 0.7149649858474731, 1.0601806640625, -0.31162574887275696, -0.7580145001411438, -0.2565491795539856, -0.7828466892242432, -0.20890751481056213, 1.3561220169067383, -0.30570119619369507, 1.1410009860992432, -1.1842241287231445, 0.3276779353618622, -0.03630969673395157, -0.6618837118148804, -0.03751785680651665, 0.21551421284675598, -0.5953649878501892, 1.3617959022521973, 0.07530467957258224, -1.2831733226776123, -1.6620248556137085, -0.6844454407691956, -0.9547751545906067, 0.9189595580101013, -0.6710183620452881, 0.897192120552063, -0.8394454121589661, -0.17939478158950806, 1.1586925983428955, 0.29195356369018555, 0.47025468945503235, -0.7324402332305908, 0.37747272849082947, -0.9038589596748352, 0.8748572468757629, 1.1176098585128784, -0.019422685727477074, -1.6037943363189697, 0.1630096733570099, 0.8977072238922119, -0.3998173177242279, -0.07910261303186417, -1.4367084503173828, 1.704036831855774, 0.5430723428726196, 0.5670189261436462, -0.11858785897493362, 1.1144546270370483, -2.12099552154541, 0.4081701636314392, -0.22132307291030884, 1.6372196674346924, -1.3497676849365234, -1.4218370914459229, -0.25054776668548584, -0.3295769691467285, 0.896569550037384, -2.0516366958618164, 0.3989403545856476, -0.6481990814208984, -0.06231505051255226, -0.12276842445135117, 2.179494857788086, 0.1523256003856659, -1.36507248878479, -1.8743383884429932, -0.4442390501499176, 2.5691683292388916, 0.3263448178768158, -0.6067237257957458, 1.0582653284072876, -1.01140296459198, -1.209800362586975, 0.09851885586977005, 0.14479538798332214, -1.6116628646850586, -1.7056993246078491, 1.4308652877807617, -0.337544709444046, 0.5247745513916016, 1.0854331254959106, -1.0614001750946045, -1.2781472206115723, -1.0534899234771729, 0.4574175179004669, 1.154928207397461, -0.5964782238006592, 0.5714492201805115, -0.9788039922714233, -0.12813033163547516, -0.23400253057479858, 0.0199015811085701], [0.06566189229488373, 0.9823886156082153, -0.5837622284889221, 0.4825133681297302, 0.006717159878462553, -0.40916457772254944, 0.11233635991811752, -0.7479186058044434, 1.418508529663086, 1.4738409519195557, -0.6257026195526123, 0.12794236838817596, 1.8718235492706299, 0.7106302976608276, 1.0026788711547852, 1.3584235906600952, 0.6664099097251892, -1.2050344944000244, 0.7465949654579163, 1.083858847618103, 0.7234677076339722, 1.8421354293823242, -1.9566067457199097, 0.06887708604335785, -1.2000833749771118, -1.158347487449646, 0.6771380305290222, -1.4935590028762817, 0.002877660095691681, 0.7634961605072021, -0.39116933941841125, 1.2208796739578247, 0.8598788976669312, 0.15595941245555878, 0.157679945230484, -1.0754903554916382, -0.8086519241333008, -1.6880985498428345, 0.24268394708633423, 0.21312810480594635, -1.4011867046356201, -1.1883848905563354, 1.4679633378982544, 2.310218334197998, -0.7978174090385437, 0.2145487517118454, 0.8071337342262268, -0.03048648312687874, -1.0634039640426636, 0.09887980669736862, -1.2324628829956055, 1.0264863967895508, 0.6642083525657654, 0.19465504586696625, 0.3301868736743927, 0.8617329597473145, -0.5141904354095459, 0.025681452825665474, 0.0925200954079628, -1.2088756561279297, -0.9396334886550903, -1.0723236799240112, 0.5957211852073669, -0.45647042989730835, -0.27604663372039795, -0.3398253619670868, 1.642820119857788, -2.2269210815429688, 0.3372373878955841, -2.4744880199432373, -1.0693459510803223, -0.5591049194335938, 0.8445087671279907, 1.4465981721878052, 0.44392848014831543, 1.4745733737945557, 0.7922787070274353, 0.4643200933933258, -0.6754891276359558, -1.0290420055389404, -1.837836742401123, -0.11168330907821655, 1.3418322801589966, 0.3204095661640167, 1.779421329498291, 0.8047055006027222, -1.5421103239059448, -1.1429792642593384, 0.17404146492481232, 0.8317531943321228, -0.6397640705108643, 1.1145169734954834, -0.5951390266418457, 2.042937994003296, 0.2112732231616974, 0.48846399784088135, -1.1553033590316772, 0.5888042449951172, -0.24562007188796997, 0.30831772089004517, -0.5344407558441162, 1.3285318613052368, -0.8845054507255554, 0.9886547327041626, 1.5222587585449219, 0.4345313012599945, 1.2428325414657593, -0.2041700929403305, -0.36272361874580383, -0.26609528064727783, 1.7054916620254517, -0.3497723340988159, 0.6730057597160339, 0.04813512787222862, 2.596071720123291, 0.14010223746299744, -0.47922202944755554, -1.345202088356018, -0.9889475703239441, 0.9243924617767334, -1.3562110662460327, 0.2978177070617676, -0.9510555863380432, 0.17643383145332336, 0.9323015809059143, 0.08487568795681, -1.490570306777954, 1.2323429584503174], [1.5985051393508911, 0.6675412654876709, 0.8954570293426514, 0.44828900694847107, 0.2356363981962204, -0.9118844270706177, -0.6869239211082458, 0.0696030855178833, -0.4274095892906189, 0.5731375217437744, 0.36718565225601196, -1.831295132637024, 0.18977123498916626, 0.6940325498580933, 0.13635464012622833, -0.7053871750831604, 0.03037753328680992, 0.2584505081176758, -0.8773932456970215, -0.9361591935157776, -0.6991045475006104, -0.0710032507777214, -3.2131404876708984, 1.0451358556747437, 1.8471988439559937, -0.41842472553253174, 0.9437156319618225, 0.24137429893016815, 1.2124371528625488, 2.682915687561035, -2.395504951477051, 0.19943886995315552, 0.02393517643213272, -0.8867297172546387, 0.9380335807800293, 0.061114389449357986, 0.2330699861049652, -0.3632524609565735, -0.11485227197408676, -0.11679185926914215, 0.31325703859329224, 1.2935302257537842, -1.9525409936904907, 0.12389253824949265, 0.46956613659858704, 0.21797296404838562, -0.7041618227958679, -0.5986332297325134, 0.8379034399986267, 0.6675257086753845, -0.7910752296447754, -0.02733132429420948, -0.3089694380760193, -2.3422276973724365, 1.7225021123886108, -0.8709638714790344, -0.5480535626411438, -1.958229660987854, 1.728871464729309, 0.4906885623931885, -0.36298245191574097, 0.794192910194397, -1.082543134689331, 2.383164167404175, 1.5001835823059082, 0.20804476737976074, -1.9406999349594116, 0.9277804493904114, -0.7109812498092651, 0.7970671653747559, 1.4462456703186035, -0.39266106486320496, -0.9943546652793884, -0.3364750146865845, 1.8027859926223755, 0.7442723512649536, -0.48625555634498596, 0.17945550382137299, -0.48477432131767273, -1.4451854228973389, -0.06024865061044693, -0.5088666081428528, -0.4854172468185425, -0.4352567791938782, 1.6156737804412842, -0.05050528794527054, 0.871599018573761, 1.3552422523498535, -0.7649535536766052, 0.17077597975730896, -0.41962289810180664, -1.5511888265609741, -0.19344158470630646, -1.2389545440673828, 1.0763590335845947, -0.45297175645828247, -1.514731764793396, 1.5268758535385132, -0.183091938495636, 1.0409985780715942, -0.30079731345176697, 0.600103497505188, -0.9128246307373047, -1.7189412117004395, 0.7741384506225586, 2.056995153427124, 0.15546022355556488, -0.48382335901260376, -2.435889959335327, -0.12744101881980896, 0.6680161952972412, 0.15058259665966034, -1.6314486265182495, 0.5022651553153992, -0.08587358891963959, 0.2168189436197281, 0.65631103515625, -0.5780384540557861, -1.4420976638793945, -1.3524467945098877, -1.4555562734603882, -0.431613564491272, -1.212770700454712, -0.5071077942848206, 0.9496694803237915, 1.2228448390960693, 0.7304155230522156, -1.8598911762237549], [0.4324844479560852, -0.792647123336792, -1.4139630794525146, -2.2459306716918945, 0.5846719741821289, -0.24267785251140594, 0.7115607857704163, 0.8766079545021057, -0.6691173315048218, -0.09644369781017303, 0.020542120561003685, -0.621563732624054, 2.000890016555786, -0.0796966627240181, -0.2682553231716156, 0.48936232924461365, -0.7815732359886169, 0.8580960035324097, -0.375966876745224, -1.5350706577301025, -1.1261874437332153, -1.9222947359085083, 2.0492215156555176, 0.6956661343574524, -0.5488405227661133, 0.09195692092180252, -0.5174410343170166, 1.3406691551208496, 1.475496530532837, -0.913922905921936, 0.7133135795593262, -0.48919424414634705, -0.7937911748886108, -1.8231197595596313, 1.6518124341964722, -2.398854970932007, 1.0367916822433472, 1.0603753328323364, -0.6300629377365112, -1.1758490800857544, -0.39416244626045227, -0.938450813293457, -0.21488195657730103, 0.749602735042572, 0.7023699283599854, -0.3973303437232971, -1.798934817314148, -1.3761123418807983, -2.257145643234253, -2.333230972290039, -0.019588707014918327, -1.0024501085281372, -0.054782744497060776, 0.17766070365905762, -1.8251301050186157, -0.08406069874763489, -0.6005679965019226, 0.44177547097206116, -0.9291331768035889, -2.1149561405181885, -1.9887685775756836, 0.16042765974998474, 0.555675745010376, 0.6847162842750549, 0.810703694820404, -1.3347439765930176, -0.6477590799331665, 1.7039477825164795, 1.8763843774795532, -1.8971794843673706, -0.10576247423887253, 1.5022153854370117, 0.9426578879356384, -1.1576392650604248, -1.0458835363388062, 0.8774108290672302, 0.706243634223938, 0.08760018646717072, 1.107447624206543, 0.0013854712015017867, 0.2703346312046051, -0.34680041670799255, 1.7122044563293457, 1.9930763244628906, -1.4701300859451294, 0.9361233711242676, -0.25684571266174316, -1.8241093158721924, -1.0592211484909058, 0.11782687902450562, 1.027502179145813, 0.2218647599220276, -0.5260317325592041, 1.4959094524383545, -0.34522897005081177, -0.44935452938079834, -0.22344039380550385, 0.4621965289115906, 0.6483927965164185, 0.2805347442626953, -0.8696252703666687, -0.2946432828903198, 0.7480164766311646, -0.7883750200271606, -1.1884747743606567, 0.3501370847225189, 0.6271457672119141, 0.9100930094718933, 1.2414441108703613, 0.08978238701820374, -1.1663906574249268, -0.8296836018562317, -1.2461857795715332, -0.9146921634674072, 0.23944009840488434, 1.0697101354599, 1.0218329429626465, -1.032107949256897, -0.21878068149089813, -1.1879483461380005, 0.4259697198867798, -2.1066768169403076, 0.5002670288085938, 1.8318206071853638, 1.2733218669891357, 1.2607629299163818, -0.7791721224784851, 0.4527552127838135], [1.1464968919754028, 0.6670555472373962, 1.1212759017944336, -0.39100342988967896, -0.5686046481132507, -0.8591768741607666, 1.4539146423339844, -0.19271086156368256, 0.014738354831933975, 0.0010125605622306466, -0.39898964762687683, -0.1928071826696396, 0.9147784113883972, 0.2608092427253723, -1.3600839376449585, -2.4964404106140137, -1.0153334140777588, 0.0950809195637703, -0.6275466680526733, -0.0692739337682724, 1.41769278049469, -0.5591451525688171, 1.0295629501342773, -0.1469242423772812, -0.008102660067379475, 1.2646832466125488, -0.055055104196071625, 0.17903529107570648, -0.45012083649635315, 2.3430871963500977, 0.751488447189331, -0.09972747415304184, 0.48401349782943726, -0.750560462474823, -1.8273489475250244, 1.246109962463379, 0.41517311334609985, 1.1029410362243652, 0.16089802980422974, -0.014651741832494736, 1.7312291860580444, 0.7518895268440247, 1.1504058837890625, -0.05235759913921356, 0.17808188498020172, -0.8730559945106506, 0.8746969699859619, 0.6245944499969482, -0.06319644302129745, 1.5998187065124512, -0.003264226485043764, -1.4525998830795288, 0.6981346607208252, -0.9051201343536377, 1.0683233737945557, -0.4892086386680603, -0.13767027854919434, -0.6320529580116272, 0.5891929268836975, 0.30759432911872864, 0.7633026838302612, 0.2465706616640091, -1.5900200605392456, -0.7057018876075745, 0.35132426023483276, -1.3304579257965088, 0.4749244153499603, -0.17832085490226746, 0.6031761169433594, 0.3070162236690521, 1.407021164894104, -0.970178484916687, 1.2374637126922607, 0.9712494015693665, -0.5846466422080994, 0.855696976184845, -1.293290138244629, -0.34172454476356506, -1.7574836015701294, -0.7146323323249817, 0.5720076560974121, -0.3483928442001343, -0.1273108422756195, 0.1255018413066864, 0.45751842856407166, 0.46559831500053406, 1.639543056488037, 2.4846253395080566, 0.12195437401533127, 0.7040272951126099, -1.164451241493225, 0.14566127955913544, 1.372004747390747, -0.4880027174949646, -0.36596354842185974, 1.022994875907898, -0.25552403926849365, 0.022077735513448715, 1.0451722145080566, 0.1642598658800125, 0.2697099447250366, 0.11735977232456207, -0.7114752531051636, 0.12120706588029861, -0.17799831926822662, 0.2779588997364044, 0.6069995164871216, -0.5566418766975403, 2.1027698516845703, -0.48756325244903564, 1.087838888168335, -0.9550240635871887, -1.2557905912399292, -0.9269726276397705, -1.4558765888214111, -0.6364315748214722, 0.9150776863098145, -0.21250572800636292, 1.6400498151779175, -0.7859843373298645, -0.9624824523925781, -0.6055914163589478, -0.38664597272872925, -1.062795877456665, 1.273759126663208, -0.21826237440109253, -0.6943802833557129, -0.6833365559577942], [0.1191035583615303, 0.12202112376689911, -0.05792318657040596, 1.5542329549789429, -1.3848732709884644, -0.1776265799999237, 0.8255395293235779, -0.646270751953125, 0.28824886679649353, 0.2791723310947418, 0.7785665988922119, 0.7195281386375427, -0.5227752327919006, -0.35750457644462585, 0.5675451159477234, 2.8218207359313965, -0.4598884880542755, -1.020370602607727, -1.1423166990280151, -0.693792998790741, 0.18220794200897217, -0.21405541896820068, -1.0835859775543213, -0.8299993872642517, 2.094344139099121, -1.0761897563934326, -0.8797637820243835, -1.616703748703003, -0.23979398608207703, 0.42846933007240295, -0.07937854528427124, -1.4016379117965698, 0.6235489249229431, -0.9460666179656982, -0.09627612680196762, -0.7123708128929138, 0.6821483373641968, 1.0143561363220215, -0.5032765865325928, 1.1760928630828857, -1.2900587320327759, -0.2109593152999878, -0.7988370060920715, -0.8012750744819641, -1.6608777046203613, -0.23047958314418793, 0.7147901654243469, -0.26000097393989563, -0.17955447733402252, -0.16139207780361176, -0.46101635694503784, 1.471369981765747, 0.06443800032138824, -0.5825226306915283, 0.7168212532997131, -0.5853408575057983, -0.5135992765426636, -0.1558127999305725, 1.199226975440979, -1.0699530839920044, 0.639892041683197, -0.852985680103302, -0.9910719394683838, 1.1058897972106934, 0.6000170707702637, -0.8647775650024414, 1.5988543033599854, -1.0670015811920166, -0.06993795931339264, 0.8080668449401855, 0.8778502345085144, 0.0434102863073349, 0.9484547972679138, 0.2138155847787857, -1.3441756963729858, -1.4821457862854004, 0.8936394453048706, -0.1170634999871254, 0.7759057283401489, 0.13283702731132507, -0.20756901800632477, 0.7642858624458313, 0.2301444709300995, 1.2950174808502197, 0.022950217127799988, -2.313843011856079, -0.1941770613193512, 0.4431821405887604, 1.187038779258728, -1.3477838039398193, -0.10104217380285263, -1.0216041803359985, -1.2753475904464722, 1.1951204538345337, -0.9439117312431335, -2.4246342182159424, -0.5658367276191711, -0.8055711388587952, 0.14758923649787903, -0.8195569515228271, -1.6171215772628784, 1.306610345840454, -1.1656988859176636, 0.6151076555252075, -1.3037573099136353, 1.3833194971084595, -1.5106796026229858, -1.2454479932785034, 0.07028304040431976, 0.6586415767669678, -0.14256875216960907, -0.8890514969825745, 0.4801122844219208, 0.01406499557197094, -0.4265094995498657, 1.2857991456985474, 2.8412253856658936, 1.079277753829956, 0.1422324776649475, -0.05374389886856079, -0.18144698441028595, 0.2126135379076004, 1.9598350524902344, 0.42862841486930847, -0.5217560529708862, 0.7627915740013123, 0.06690862774848938, -0.4350057542324066], [1.0856409072875977, 0.0979846939444542, -0.40129807591438293, -0.8161816000938416, 0.15577849745750427, 0.7479727268218994, 2.437509059906006, 0.5990975499153137, -0.2536923587322235, 0.33802467584609985, -2.222991466522217, -0.3151077330112457, -1.0136317014694214, 1.3244948387145996, 0.10115023702383041, 1.2071703672409058, 0.47735655307769775, 1.963185429573059, 0.19915851950645447, 0.48708871006965637, -1.2473148107528687, -1.3691866397857666, 0.10169260203838348, 0.1842227727174759, -0.4723515510559082, -0.7692748308181763, -0.16852784156799316, -0.3436146676540375, 0.47471460700035095, -0.6508891582489014, -0.5030545592308044, 1.616377830505371, -0.8685182332992554, 1.0052013397216797, -0.4842222034931183, 0.2156795859336853, -0.7868064045906067, 1.8115918636322021, -1.6780662536621094, 0.10802211612462997, 0.5938643217086792, -2.1942367553710938, 0.027790384367108345, 0.4728771150112152, -0.11978999525308609, 0.8432317972183228, -0.8324607014656067, -0.9737804532051086, -1.551377534866333, -2.0386276245117188, 1.0868943929672241, -1.4444708824157715, -2.6401994228363037, -0.859987735748291, 0.22902750968933105, 0.30998530983924866, 0.7798300385475159, 0.6944396495819092, -0.03249664232134819, 0.5046732425689697, -0.4487013816833496, -0.5893266797065735, 0.4565228223800659, -2.0101184844970703, -0.626681923866272, 0.8745743632316589, -0.7512755393981934, -1.3383047580718994, -1.2399840354919434, -1.97858464717865, -2.229562520980835, -0.5823709964752197, 1.1039652824401855, 0.8864845633506775, 2.0423781871795654, -0.1802673041820526, -0.4526301622390747, 0.5912826657295227, 0.09354996681213379, 0.03276621922850609, -0.3079793453216553, -1.021004319190979, -0.46714040637016296, -0.5056672692298889, -1.6436452865600586, 2.4821863174438477, 0.7501358389854431, 0.37258023023605347, 0.29603782296180725, 1.4836502075195312, 2.211568832397461, -0.2790719270706177, 1.3293150663375854, 0.5965914726257324, 1.1713776588439941, 0.1919826716184616, -1.9667900800704956, -0.6801121234893799, 0.09160242229700089, -0.2362155169248581, 3.126328229904175, 0.7846126556396484, -0.3267998993396759, 0.05593560263514519, 1.4859542846679688, -0.9590994715690613, -1.321641206741333, -0.4547581970691681, 1.1102274656295776, -0.22429490089416504, 0.8469089865684509, 1.5101442337036133, 1.7746901512145996, -0.22129693627357483, 1.8193449974060059, 0.3505043089389801, -1.2027510404586792, 1.4323015213012695, 0.6752868890762329, -0.3536672294139862, -1.019453525543213, 0.036054957658052444, 0.6112858057022095, -0.22679071128368378, -0.22988811135292053, -0.7360263466835022, -0.23514018952846527, 0.334099680185318], [1.6723819971084595, -0.37713247537612915, 1.0276466608047485, 0.6802226305007935, -0.5131598114967346, 1.4600236415863037, 1.2191435098648071, 0.2397104799747467, 1.998529314994812, -0.7193592190742493, -0.33501437306404114, 1.8480581045150757, -1.6455130577087402, -0.02518981695175171, 1.0220104455947876, 1.0186822414398193, 0.1976354569196701, 0.9193295240402222, -0.4419349133968353, -0.35157930850982666, 0.3978557288646698, 0.7822408080101013, -0.7378278374671936, -1.124495506286621, 0.2316892296075821, 0.07662545889616013, 0.6998704671859741, -0.2952214181423187, -1.3542020320892334, -0.8079032301902771, -1.6589659452438354, -0.035262331366539, 0.6819707751274109, -0.09525739401578903, -0.29719600081443787, -0.85938960313797, 1.300847053527832, 0.4176872670650482, -0.1426277458667755, -1.5612516403198242, -0.34506872296333313, 0.31926172971725464, 0.300008088350296, -1.2182655334472656, 0.01295387838035822, 0.4740768373012543, -1.2409512996673584, -1.2303955554962158, -1.0212626457214355, 1.134512186050415, 0.65083909034729, -0.9468206167221069, -0.6001918315887451, 1.60446035861969, 0.421286016702652, 0.4089601933956146, 0.40335577726364136, 1.8682276010513306, 0.4386291205883026, -1.0293077230453491, -1.0452886819839478, -0.7533079385757446, 2.4912707805633545, 0.9674222469329834, -0.4103565216064453, -1.2755072116851807, -0.7491098046302795, -0.2845141589641571, 0.1079685166478157, 0.1999536007642746, -0.5703862309455872, 0.0771414265036583, -1.0373798608779907, -0.48071035742759705, -1.2123761177062988, -0.43393656611442566, -0.6214084625244141, -0.5487681031227112, 1.2033812999725342, 0.5234420895576477, 0.6485452055931091, 0.3287648558616638, 1.671348214149475, -0.3576481342315674, 0.7033805251121521, 0.5768378973007202, 0.1340089738368988, 0.15057753026485443, -1.4445269107818604, 1.2559670209884644, 2.903491735458374, 0.47888094186782837, -0.2765217423439026, -1.2517579793930054, 2.2552690505981445, 1.2073355913162231, -0.3453463613986969, 1.529402256011963, -0.33642613887786865, -0.6903754472732544, 1.0186406373977661, 1.537923812866211, -0.09327894449234009, -0.5120529532432556, -0.7423936128616333, 1.1170434951782227, 0.03771233931183815, -0.9346361756324768, 1.6404236555099487, -1.1327171325683594, -1.6503033638000488, -1.1021513938903809, -0.6085623502731323, 0.32117998600006104, -0.5455093383789062, -1.3968507051467896, 0.09220618009567261, 0.9683723449707031, 1.7183337211608887, 0.1973503679037094, 1.8438267707824707, 0.040884315967559814, -1.9631245136260986, 0.041105397045612335, -0.22408263385295868, 0.6598954200744629, 1.3469338417053223, -0.3298937976360321], [0.5492904186248779, -0.7934813499450684, 0.8663407564163208, 0.39002519845962524, -0.6888181567192078, -0.5757762789726257, -0.6032971739768982, 1.8554012775421143, -0.8205080628395081, -0.2222982496023178, 0.2519562244415283, -1.9494038820266724, -1.6601245403289795, 1.3344523906707764, -0.60368412733078, 0.27996206283569336, -0.06312218308448792, 2.5515685081481934, 1.0076874494552612, 0.09514518082141876, 0.8900032043457031, 1.284271478652954, 0.536358654499054, -1.6022274494171143, -1.680594801902771, -0.6317634582519531, 0.3804241120815277, 1.3911691904067993, -0.5381540060043335, -1.132228970527649, 0.4241371750831604, 0.265956312417984, 0.4631134569644928, 0.21037374436855316, 1.7176713943481445, 0.5257721543312073, -0.8626726269721985, -1.494261622428894, -0.10268035531044006, -1.0924007892608643, 0.2069319188594818, -1.6052076816558838, -0.13349959254264832, 0.9426639080047607, 0.017932403832674026, 0.9466490745544434, -0.5942659974098206, -0.7666924595832825, 0.09712129086256027, 0.3078746199607849, -1.3984644412994385, -0.3923301696777344, 0.14180819690227509, 0.3833523392677307, -1.1561367511749268, 0.5300725102424622, 0.028236234560608864, 0.5442771911621094, -0.4827767610549927, -0.9576355218887329, 0.8006616830825806, -0.9237334132194519, 0.62374347448349, -1.2975959777832031, -0.18631954491138458, 1.0157464742660522, 0.10992055386304855, -2.117243528366089, 1.3792333602905273, -0.6631077527999878, -0.0048398845829069614, -0.5284528136253357, 0.10741081833839417, -0.4502618610858917, 0.39738011360168457, 1.4612665176391602, 0.24495922029018402, -1.6583638191223145, -1.1677677631378174, -0.06616737693548203, -1.1271722316741943, 0.4774037301540375, -2.8251307010650635, 0.42789897322654724, -0.7341794371604919, -0.9734293818473816, 1.97171151638031, 0.9204210042953491, -1.0402098894119263, 1.6656020879745483, 1.2069547176361084, -1.380278468132019, -0.2087346762418747, -0.08291787654161453, -0.043309878557920456, -0.9739804267883301, -1.4077719449996948, -0.040953803807497025, 0.6834862232208252, -0.15793383121490479, 0.5667532086372375, 0.6897429823875427, 1.0192384719848633, 0.6732936501502991, -1.2812117338180542, -1.5850683450698853, -1.0098026990890503, 0.6184379458427429, -1.6488693952560425, -0.5565758943557739, -0.8202366232872009, -0.3732476234436035, 0.6145372986793518, 1.4626799821853638, -0.5576091408729553, 1.5845365524291992, 0.5437525510787964, -0.5707741379737854, -1.610092043876648, 0.4424714148044586, -0.7034303545951843, -0.19128936529159546, 0.1731709986925125, -0.38564950227737427, 0.11081410944461823, -0.16882435977458954, 0.3043248653411865, -0.7656900882720947], [0.02758481167256832, 0.2148396074771881, 0.07288787513971329, -0.9127931594848633, 1.9709341526031494, -0.8570075631141663, -3.3839848041534424, 0.6964888572692871, -0.934733510017395, 1.542377233505249, 0.05389128997921944, 0.7431786060333252, 0.6660935878753662, -1.3761324882507324, 0.03895297273993492, -0.13026268780231476, -1.2623043060302734, 2.0424182415008545, -0.8440884947776794, -1.4263639450073242, 0.7362164258956909, 1.503461480140686, 1.5580735206604004, 0.5235539078712463, -1.7125965356826782, -0.07205090671777725, -0.6282748579978943, -0.584479808807373, -0.13422593474388123, -1.8069674968719482, 0.5089566111564636, -1.5380946397781372, -0.6260656118392944, 1.490670084953308, 1.293088674545288, 0.5117726922035217, -0.6905786395072937, 2.0260608196258545, 0.8006495237350464, -1.1236492395401, 0.9915874600410461, 0.3310115933418274, 0.7841216921806335, -0.15736794471740723, 0.5826088190078735, 0.34031444787979126, -0.5323930382728577, -1.065563440322876, 0.11059215664863586, 1.518836498260498, -0.5442919135093689, 0.6164247989654541, -1.4973920583724976, -0.09670208394527435, 1.8181978464126587, 2.1003293991088867, -0.32839131355285645, -0.6078751087188721, 0.43959903717041016, 1.6412162780761719, 0.6008890867233276, 1.0151995420455933, -0.6966211795806885, -0.0854206457734108, -0.6646965146064758, -1.7354809045791626, 0.4855835735797882, -0.36422088742256165, -0.5594303011894226, 0.004002981353551149, -0.3877575397491455, -0.6778295040130615, -0.4031832218170166, 0.34006914496421814, 0.62713223695755, 0.279251366853714, -0.9234368801116943, 0.6548978686332703, -0.14518198370933533, -0.2416745275259018, -0.8223685622215271, 0.15671798586845398, 0.36249300837516785, -1.2894086837768555, 1.5183494091033936, -0.9987421035766602, -0.4691011905670166, 0.30854952335357666, -0.5843976140022278, -1.6919264793395996, -2.05224347114563, 0.3093855082988739, -0.5582367181777954, 0.8362981677055359, 1.082542061805725, 0.045520808547735214, -1.116193413734436, -1.0515480041503906, -1.6484310626983643, 0.8244467973709106, 0.23496897518634796, -0.611035168170929, 1.8447620868682861, 1.3443653583526611, -0.5073627233505249, 1.6214776039123535, 0.40251022577285767, -1.06549072265625, -0.2953190207481384, -0.39553359150886536, -0.4677228629589081, -0.03317795693874359, -0.05068294331431389, -0.6268194913864136, 1.3707512617111206, 1.6690523624420166, -1.542679786682129, -0.10155239701271057, -0.058088503777980804, 0.26985445618629456, -1.5966427326202393, -0.7754957675933838, 0.656367838382721, -0.4544074535369873, 0.8723391890525818, -0.9294787645339966, 1.9886964559555054, -0.8283749222755432], [-1.5298362970352173, 1.009442687034607, 0.9524197578430176, -0.609268069267273, -2.2577006816864014, 1.6041969060897827, -0.6767407655715942, 0.20020602643489838, -1.1764122247695923, -0.5279743075370789, 0.353133887052536, -0.7725990414619446, 0.9549540877342224, 0.4043748378753662, -0.10308258980512619, -1.8630539178848267, -0.09712066501379013, -0.3916275203227997, -1.0878616571426392, 1.0127886533737183, -1.9647597074508667, -0.7622861266136169, -0.8539232611656189, 1.1838994026184082, -0.9345760345458984, 0.7651927471160889, -0.37515491247177124, -1.2206037044525146, 2.310729503631592, -0.6915726065635681, -0.9397198557853699, 0.40230804681777954, -0.373053640127182, 0.5873727798461914, -0.2613333463668823, 0.7965426445007324, -1.3561162948608398, -2.1681745052337646, -0.4518383741378784, -0.1661255806684494, 0.5754691958427429, 0.3878859281539917, -1.1017850637435913, 0.16144497692584991, 1.2365063428878784, -0.10767384618520737, 1.1435848474502563, -1.2688579559326172, 0.8785127401351929, 1.30637526512146, 1.0250194072723389, -0.5968165397644043, 0.762134850025177, 0.32449355721473694, -0.6661900877952576, -0.043152984231710434, -0.8956700563430786, -1.2780920267105103, -0.1249890848994255, 1.0225614309310913, -0.5135709047317505, 1.271405577659607, -0.09444929659366608, 0.5489454865455627, 0.4680451452732086, 1.2663756608963013, -0.09650915861129761, -0.9854211211204529, -1.2848458290100098, 0.7232807874679565, -0.9441748857498169, -0.7861281633377075, -1.5751551389694214, -0.47598281502723694, -1.2444829940795898, -0.4586172103881836, 0.6980727910995483, 0.882262647151947, 1.184733510017395, 1.0164791345596313, 0.6296395659446716, -0.43084946274757385, 3.346230983734131, -0.7568998336791992, -0.6234286427497864, 0.11605000495910645, 0.19592367112636566, 1.3910492658615112, -0.512730598449707, 1.2302007675170898, -0.6906545162200928, -0.8783622980117798, -1.0965367555618286, -2.025907278060913, -0.13998469710350037, -1.552544355392456, -0.0159518551081419, -0.7435780167579651, 0.22888846695423126, 0.7231948375701904, 0.6857719421386719, -1.1430484056472778, -0.5568716526031494, 1.037034273147583, -0.0520659014582634, -0.5023939609527588, 0.6273663640022278, 0.5799707770347595, -0.10364396870136261, 0.23115696012973785, 0.20209096372127533, -0.0481293611228466, 0.266635000705719, 0.2633019983768463, 0.15045809745788574, 0.07662805914878845, -0.768445611000061, 0.8765341639518738, 0.1484232395887375, -1.3496302366256714, -1.3004950284957886, -1.5191972255706787, 0.9926650524139404, -0.26111701130867004, 1.0568997859954834, -0.05073792114853859, -0.42285770177841187, 0.45612120628356934], [0.6396188735961914, -0.42963898181915283, -1.085890531539917, -0.481680303812027, 0.3480980098247528, 0.32618939876556396, 0.4605219066143036, -1.2298322916030884, 0.15753191709518433, -1.3466554880142212, 0.825116753578186, 0.22394534945487976, -0.17626236379146576, -0.44798392057418823, -1.12446928024292, -0.9533511400222778, -0.24319246411323547, 0.4471683204174042, 0.30966562032699585, -1.5006784200668335, 0.6909112930297852, 0.47545337677001953, -0.6374252438545227, -1.032960057258606, 0.16802483797073364, 2.052281379699707, -0.9993624687194824, -0.024241480976343155, -0.14918822050094604, 0.9741335511207581, 0.7950755953788757, -0.1211291030049324, -0.18445204198360443, 1.559016466140747, 0.6828076243400574, 0.5306652784347534, -0.2896920442581177, 0.5268494486808777, 0.35018450021743774, 0.5608544945716858, -0.10674510896205902, -0.44823628664016724, -0.05779971554875374, 0.7478899955749512, -0.23067502677440643, 0.15368318557739258, -1.336710810661316, 0.271857351064682, -0.06521426886320114, 1.373846411705017, -0.4263056516647339, 0.7801476120948792, 0.3585864007472992, -0.37572017312049866, -1.3108073472976685, -1.1294801235198975, 0.5390967726707458, 0.9450000524520874, -0.02222595363855362, -0.6488404870033264, 0.1670527309179306, -0.5809874534606934, -0.37802088260650635, 0.4000273048877716, -0.23499982059001923, 0.8776863217353821, 0.08975869417190552, -0.8318321108818054, -1.2209417819976807, 1.6567456722259521, 0.7544595003128052, -0.6814244985580444, -0.04264746233820915, -0.5875461101531982, 0.4722743630409241, -0.2830774784088135, 1.2293545007705688, 1.0495356321334839, 0.6579853296279907, 0.537284791469574, -1.5234956741333008, -0.18472889065742493, -0.4213477671146393, -0.35747653245925903, -0.7943642139434814, 0.5552070140838623, 0.8135530948638916, -0.25952836871147156, 1.5640653371810913, 0.35889488458633423, -0.09596216678619385, 0.3503522276878357, -0.15787222981452942, 0.11593764275312424, 0.42224881052970886, -1.0442728996276855, 1.3218580484390259, 0.5453274846076965, -1.1344029903411865, -0.2768356502056122, 2.0513417720794678, 1.9014184474945068, -0.8754458427429199, 0.12388355284929276, -1.2139290571212769, -0.766314685344696, 0.24348624050617218, -0.5876122713088989, 0.1677943468093872, -1.8995248079299927, 1.0444046258926392, 1.684037208557129, 2.4010376930236816, 0.22790510952472687, 0.5274595022201538, 0.7063463926315308, 1.2894785404205322, -0.5649385452270508, -0.45559072494506836, -0.5664196610450745, -0.6776303648948669, 0.810187041759491, -0.31676921248435974, 0.020215261727571487, 1.4692965745925903, 0.861644983291626, 0.7201904058456421, 0.968985378742218], [-1.78742516040802, -0.23470938205718994, 0.4367963969707489, -1.1436278820037842, -0.1949075609445572, 1.4872989654541016, -0.8872644305229187, -0.6733525395393372, 1.5060827732086182, -0.9529366493225098, 1.2239878177642822, -1.559812307357788, -0.15342696011066437, -0.22692547738552094, -0.0644451454281807, 0.6496615409851074, -0.21202954649925232, 0.5549537539482117, -2.0016863346099854, 1.542999505996704, 0.1719866693019867, -0.8959117531776428, -1.1301751136779785, 0.5847247838973999, -1.817754864692688, -1.5236623287200928, 1.060708999633789, -0.16940419375896454, 0.7706347107887268, 0.06237049028277397, 0.3538842797279358, 0.6187134385108948, 1.0303655862808228, -1.7166575193405151, 1.8434425592422485, 0.8000326752662659, 0.9933235049247742, 2.4448516368865967, -0.7218142747879028, 1.3009002208709717, 0.5578927993774414, 0.6646672487258911, 0.2029067724943161, -0.6413428783416748, -0.08767463266849518, -0.9654606580734253, 1.2076143026351929, -0.7396605610847473, -0.5585584044456482, -0.674197256565094, -1.161850094795227, -0.5827890038490295, 0.38304999470710754, -0.09324712306261063, 0.6008724570274353, -0.43212392926216125, 1.0671846866607666, 1.558323621749878, 0.3887435495853424, -0.26132550835609436, 0.22634148597717285, -0.26091235876083374, -1.5674843788146973, -1.720828652381897, -0.2308085560798645, -1.7743867635726929, -0.9254441857337952, -0.7600926160812378, -0.39518842101097107, 1.3379791975021362, 1.060379981994629, -1.6063512563705444, -0.7110645771026611, -0.4037911593914032, -0.8317460417747498, 0.0725339725613594, -2.756913185119629, 0.18300314247608185, 0.44094017148017883, 0.5212404131889343, -1.4370895624160767, 0.19187642633914948, 0.029987197369337082, -0.39981159567832947, -1.00594961643219, -2.4039218425750732, 0.0018172255950048566, -1.9869384765625, 1.5077074766159058, 0.6935285925865173, 0.5003361105918884, 0.4791480004787445, 0.7398269772529602, -1.180895447731018, 0.03614022955298424, -0.6816936135292053, 1.4365088939666748, -1.2673990726470947, -0.7799620032310486, 0.3610839545726776, -0.515679121017456, -0.2757035493850708, 0.9271023273468018, -0.307255357503891, 0.2968699634075165, -0.6255141496658325, -1.2772570848464966, -0.26672816276550293, 2.4068424701690674, -0.8517418503761292, -1.133163571357727, 0.003219055477529764, 0.2333465963602066, -0.484027624130249, -0.5324606895446777, -1.1766752004623413, -0.3212079703807831, -0.05737810581922531, -0.6812156438827515, 0.010861585848033428, -0.16924896836280823, 1.780617356300354, -1.318698763847351, 1.8948160409927368, 0.41636890172958374, -0.43055006861686707, -0.09145709872245789, -1.86543607711792], [1.5059453248977661, 0.039868369698524475, -0.8665314316749573, -0.12262915819883347, -1.1438928842544556, -0.8552184104919434, -0.9025365710258484, -0.7028882503509521, -1.4282786846160889, 0.30604052543640137, -0.3450319766998291, 2.4889516830444336, -0.7773941159248352, -0.6738408803939819, -1.213482141494751, -0.6050442457199097, 0.4640214443206787, 0.866920530796051, -0.9533358216285706, 0.6770533323287964, 0.37185853719711304, -0.13300208747386932, 0.2922287881374359, 0.027112461626529694, 0.6683028340339661, 0.6501584649085999, -0.2513916492462158, 0.33522719144821167, 1.1002193689346313, 0.4974689483642578, 0.9856283068656921, 0.4036930501461029, -1.0515997409820557, 2.0771284103393555, -1.8476570844650269, 1.3798586130142212, -0.597511887550354, -0.8596517443656921, -0.03907882422208786, -0.788718581199646, 0.16136161983013153, -0.381520539522171, 0.5254825353622437, -0.1122363731265068, 1.3480156660079956, 0.9505837559700012, -0.1305207759141922, 2.1116080284118652, -1.0064635276794434, 0.5977064371109009, -1.9108248949050903, 0.659932553768158, -0.9661815166473389, -0.09567391127347946, 0.31155839562416077, -0.33715376257896423, -1.400099277496338, -0.4884832203388214, 0.1654074639081955, -2.354094982147217, -0.36492621898651123, -0.8634868264198303, -0.8681281805038452, -2.1289443969726562, 0.31032028794288635, -1.1864497661590576, -0.36481261253356934, 1.4558191299438477, -0.3965704143047333, 0.8180150389671326, 0.424242228269577, -0.5335540175437927, 0.11257299035787582, 0.5267482399940491, -1.9650849103927612, -1.937766671180725, -0.22162096202373505, 0.956577479839325, 0.004395271185785532, 0.742620050907135, 1.4164916276931763, -0.5598992109298706, 0.1741897314786911, -0.6266056299209595, 0.1310749053955078, -1.1495925188064575, 1.8523530960083008, 0.35426434874534607, -1.4807963371276855, 0.924457311630249, -0.5679517388343811, -0.40081775188446045, -1.204192042350769, 0.48819398880004883, 0.6491689682006836, -0.3445475995540619, 0.5393393039703369, 0.4132494330406189, -0.33444568514823914, 0.9880531430244446, 1.375691533088684, 1.1262660026550293, 0.19932924211025238, -0.5275534987449646, -0.08668737858533859, -0.13222502171993256, -0.8243232369422913, 1.0408730506896973, -2.4406909942626953, 0.5753656625747681, 1.3665891885757446, -1.2149244546890259, -0.7648742198944092, 1.3558708429336548, 0.7598811984062195, -0.6402469873428345, -1.770959496498108, -0.9977108836174011, 0.5755562782287598, -1.7233803272247314, 0.009407952427864075, 0.1787707805633545, -0.4312717616558075, -2.305595874786377, -0.1709422916173935, 0.28220823407173157, -1.057071566581726, -0.3191598355770111], [0.7896985411643982, -0.08162444829940796, 0.9454213976860046, -1.5515038967132568, 2.162351608276367, -0.9102132320404053, -0.14549900591373444, 1.1141248941421509, -1.2111029624938965, 0.2571883499622345, 1.3793504238128662, -0.7199987769126892, 0.30451586842536926, 0.8371661901473999, -1.2995208501815796, -1.2234148979187012, -0.5141778588294983, -0.2542330026626587, 0.308948278427124, 0.14853303134441376, -0.7409741282463074, 0.6769601106643677, -0.6218489408493042, -0.18399593234062195, -0.2672884166240692, -1.1132243871688843, 0.411973237991333, -1.0019451379776, -1.631604790687561, -1.353651762008667, 0.57417893409729, -0.5426466464996338, 0.5095959305763245, -0.4597151577472687, 0.8775494694709778, 0.7224586606025696, -0.3840128779411316, 0.8581092357635498, 0.3271197974681854, -0.5580608248710632, -1.0482497215270996, 0.07894671708345413, 0.9028684496879578, 0.7087423801422119, 0.5263793468475342, -0.12935571372509003, -0.4081045985221863, -0.22293387353420258, -1.4822989702224731, -0.9298322200775146, 1.2685496807098389, 1.3604098558425903, 0.8399062752723694, 1.1555414199829102, -1.2812830209732056, 0.8796479105949402, -0.12261554598808289, 1.5726613998413086, -0.7290248274803162, 0.7851109504699707, 0.3820783793926239, 0.6816566586494446, -0.018449651077389717, -0.6877314448356628, -0.2787967026233673, -0.8313582539558411, 0.8792135119438171, 0.08944275975227356, 0.954407274723053, -0.36989715695381165, 0.2276267111301422, -0.1872863918542862, -1.5263601541519165, -1.5734105110168457, 1.0233319997787476, 3.1863858699798584, -1.3601776361465454, 0.023235663771629333, 0.32743093371391296, 1.5852185487747192, 2.116384983062744, 0.38776686787605286, 1.6296253204345703, 0.15379014611244202, -0.5541000962257385, -1.2924034595489502, -0.6109440922737122, -0.6859108209609985, -0.8390633463859558, 1.1869617700576782, 1.5194957256317139, -1.572033166885376, -0.812411367893219, -1.7918603420257568, -0.8847068548202515, 0.9969789981842041, -0.014594310894608498, -0.4605356454849243, 2.3584158420562744, -0.9551058411598206, 1.0264806747436523, 1.3769069910049438, -0.8044055104255676, 0.8086786866188049, 1.7269761562347412, 0.3762187659740448, 0.23945721983909607, 1.1654199361801147, 1.0909768342971802, -1.3636798858642578, 1.0081562995910645, 0.6759835481643677, 0.12937010824680328, -0.049227677285671234, 1.3411067724227905, -0.9386646151542664, 1.6988624334335327, 1.0114233493804932, -0.12801826000213623, -1.3727936744689941, -1.3258767127990723, -0.20486465096473694, -0.15431800484657288, -0.5271247029304504, 0.036941248923540115, -0.9930267333984375, 0.8358432054519653, -1.014023780822754], [0.729263186454773, -0.0016983902314677835, 0.8065717816352844, -0.18338528275489807, -0.3625558912754059, -0.883499264717102, -0.7161645293235779, -0.3619851768016815, 0.8201950788497925, 1.6558647155761719, 1.2373323440551758, 0.2566966712474823, -1.406363844871521, -1.337231159210205, -0.6995762586593628, -0.7471082210540771, 0.2553918957710266, -0.4785539209842682, -1.2512685060501099, 0.6356698870658875, 1.3020564317703247, 1.1148386001586914, 1.8363957405090332, 1.169622540473938, -0.9858670830726624, 0.35471853613853455, 1.148103952407837, 0.609975278377533, 1.4488182067871094, -0.0311888474971056, -1.3473131656646729, -0.7784854173660278, 1.253343939781189, 0.2060365080833435, -0.5026068091392517, 1.5019985437393188, 0.6711028218269348, 0.002594341989606619, -0.3921181261539459, -0.5742980241775513, -0.33492475748062134, -2.0492751598358154, -0.7184592485427856, -1.2124645709991455, 0.7383723258972168, 0.14023829996585846, -0.1969347894191742, 0.10929175466299057, 0.9984004497528076, 0.16485989093780518, -1.5035243034362793, 0.3762781620025635, 0.6837145090103149, 0.6555584073066711, 1.9546167850494385, 2.3605122566223145, -1.5971646308898926, 0.5967397093772888, -0.6441986560821533, 0.2096739411354065, 1.108336329460144, -1.034043550491333, -2.535015344619751, -1.5337966680526733, -0.2079869955778122, -0.9716155529022217, -1.1737127304077148, 1.402526617050171, -1.3268375396728516, 1.7243951559066772, -0.26208072900772095, 0.6485638618469238, 0.5695523619651794, -0.3353867828845978, -0.04589123651385307, -0.5617597103118896, 1.8328043222427368, -0.3791161775588989, -0.7401520013809204, 1.409869909286499, 0.734869658946991, 0.2888696789741516, -0.32124975323677063, -0.2384178191423416, -0.07045429944992065, -0.3583816885948181, -0.8804309964179993, 1.3264636993408203, 0.5456516742706299, 1.1796128749847412, 0.7400645017623901, 0.24881581962108612, -0.6897000670433044, 0.34242796897888184, 0.4257976710796356, -0.29578378796577454, 0.08967756479978561, 0.6569899916648865, -0.9615516662597656, -0.6227359175682068, -1.1991822719573975, 0.013421501964330673, -0.5062472820281982, 0.8465662002563477, -0.937239944934845, 0.8794440031051636, -1.9990031719207764, 0.5692781805992126, -0.2558612823486328, 0.9275676608085632, -0.5128101706504822, -0.7927919626235962, 1.3965493440628052, 0.18978402018547058, -1.5396571159362793, 0.4762628674507141, -0.4685068726539612, 0.746422529220581, 1.2875632047653198, -1.387963056564331, 0.284572958946228, 1.5710221529006958, 0.4334150552749634, 0.12410955876111984, 0.5564892888069153, 0.006351541727781296, -0.01136485580354929, 1.4170095920562744], [-0.06749938428401947, -0.4783070683479309, -0.6257086396217346, -1.2042243480682373, -1.2027708292007446, -0.527532696723938, 1.1724900007247925, -1.615982174873352, 0.24894727766513824, 1.4422316551208496, 0.4545348882675171, 0.7591071128845215, 2.8934619426727295, 1.860055923461914, -1.7415480613708496, -0.396594375371933, -0.9344667792320251, -0.24702313542366028, 1.3805456161499023, -1.7897286415100098, -0.1848922073841095, 0.5591942667961121, -0.07799845933914185, -0.7439844012260437, 1.086154580116272, -0.8925167322158813, 0.20800678431987762, -0.14606541395187378, 2.19326114654541, 0.2917481064796448, 0.30624741315841675, 1.7054580450057983, 0.09407338500022888, -0.17278902232646942, -1.6979050636291504, -0.7581697106361389, 0.7571828961372375, -0.43567514419555664, -1.9472265243530273, -0.378018319606781, 0.14072559773921967, 0.9540719985961914, 1.5627470016479492, -0.012130506336688995, -1.4372520446777344, -2.0630388259887695, 0.7188668847084045, -0.03098302148282528, 0.7170552015304565, -1.7367758750915527, 0.7869026064872742, 0.8907056450843811, -0.7019484639167786, 1.901402473449707, 0.9314496517181396, 2.3905107975006104, 0.5260595083236694, -0.5157217979431152, 0.034717295318841934, -0.3499525785446167, -0.23182162642478943, 0.4935588538646698, 0.6815164685249329, 0.6005849838256836, -1.6547269821166992, -0.24095064401626587, 0.3554975092411041, -1.4212148189544678, -0.5174816250801086, -0.61541748046875, -0.4883207082748413, 0.781671404838562, 0.736031711101532, 0.33700692653656006, 1.4205914735794067, 0.1612202227115631, 0.7242408394813538, 1.129501223564148, -0.748478353023529, 0.7569625973701477, 0.5064489841461182, 0.30150556564331055, 0.021887371316552162, 0.7174223065376282, -0.08284314721822739, -0.08023486286401749, -1.5157475471496582, -0.7668848633766174, 0.5776873230934143, -0.2741281986236572, 0.567535400390625, 0.7098727226257324, 0.5086765289306641, 0.7698103189468384, -0.2204742282629013, -0.5805498957633972, -0.9385620951652527, -0.032828494906425476, -0.9127072691917419, -0.8254047632217407, 0.8484931588172913, 1.0040397644042969, -0.06267672032117844, 0.557529628276825, -1.1376689672470093, 0.862786054611206, -0.8746211528778076, 0.7605767250061035, 0.47607576847076416, 1.050297737121582, -1.4202643632888794, -0.32471075654029846, -0.38579046726226807, 0.44728517532348633, -0.09333071112632751, 1.054598093032837, 0.3164886236190796, 0.3626829981803894, 0.3153693974018097, 0.10080995410680771, -1.5070136785507202, 1.0082260370254517, -0.6072859168052673, 0.787807285785675, -1.2661521434783936, 1.0619916915893555, -1.7228399515151978, 1.2641818523406982], [1.194467544555664, -0.32449957728385925, -0.698631763458252, -0.00800045020878315, 0.017460277304053307, -0.47595733404159546, 1.37395179271698, -0.4088101089000702, -0.29789894819259644, 1.0406416654586792, 1.2945083379745483, 0.6481435298919678, -0.5011442303657532, 0.1137128695845604, 2.414783239364624, 0.8125231266021729, -1.3741331100463867, -0.406270295381546, -0.11649728566408157, -2.560922384262085, -0.7887495160102844, 0.9211614727973938, 0.5219019651412964, 0.22401714324951172, -0.08129967749118805, 0.6988741159439087, -1.817646861076355, -0.9930893182754517, -0.1813984513282776, 0.6641094088554382, -1.270339012145996, -0.2513595223426819, 0.15050281584262848, 0.409736305475235, 0.9847484827041626, 0.6768029928207397, -0.8678854703903198, 0.5024005174636841, 0.19865034520626068, -0.07776237279176712, 0.7130621671676636, -2.0383920669555664, -0.1537129282951355, -0.9247813820838928, 0.4658443033695221, -2.012641429901123, -0.6255871653556824, -0.11735700070858002, -2.2290091514587402, -0.30163314938545227, 1.0682544708251953, -0.10334967821836472, -0.34142524003982544, -1.1296889781951904, 0.8451225757598877, 0.44704243540763855, -1.1482000350952148, 1.727577805519104, 1.2317934036254883, -0.6602336764335632, 0.1510561853647232, 1.3026230335235596, -0.6268664002418518, 0.7651921510696411, 1.3902673721313477, -0.39931946992874146, -0.7549911737442017, -0.5028098821640015, 0.22196835279464722, 1.216048240661621, 0.6061100959777832, -0.7910786271095276, 0.10813792049884796, 1.812287449836731, -2.5460689067840576, -0.13918817043304443, 0.511612057685852, 1.0807464122772217, 0.15251979231834412, -0.49969372153282166, 0.3570246398448944, 2.584465503692627, -0.27190014719963074, -0.345385879278183, -2.1907384395599365, 1.0076572895050049, -0.32759881019592285, -0.22122685611248016, 0.9063054323196411, 0.9353286623954773, 0.518304169178009, 0.045841947197914124, -0.8868184089660645, 0.40971583127975464, 0.7527781128883362, 1.0211845636367798, 1.274446725845337, -0.44827404618263245, -0.8990836143493652, -2.0089404582977295, -0.24147431552410126, -0.759048342704773, -0.6068650484085083, 1.613957166671753, 1.0185601711273193, 0.10618285834789276, 0.9554566740989685, -0.4042578339576721, 1.3662567138671875, -0.5576445460319519, 0.021477054804563522, -0.002384069375693798, 0.9802230000495911, 1.1429309844970703, 0.40389788150787354, 0.24728280305862427, -0.2477055937051773, -1.026302695274353, 1.5880694389343262, 0.6522030234336853, 1.0387171506881714, -1.230589509010315, 0.19155466556549072, -0.6418771147727966, 0.8909258842468262, -0.6493650078773499, -0.33339351415634155, -1.433189868927002], [0.6188638210296631, -0.21193905174732208, -1.7187392711639404, -0.36206746101379395, -0.40846848487854004, 0.7143807411193848, 0.04750942811369896, 0.6473219990730286, 0.38198256492614746, 1.8675254583358765, 0.9607875347137451, 0.3077095150947571, -0.7636480927467346, -0.0258137546479702, -1.6374638080596924, -1.3563313484191895, 0.920420229434967, 0.19802235066890717, 0.7056212425231934, -1.1695244312286377, 0.12074434012174606, -0.17003168165683746, -0.9841856360435486, 2.0052547454833984, 0.08335884660482407, -0.463495671749115, -1.4186925888061523, -1.1052074432373047, -0.009470540098845959, -0.8194776177406311, 0.7972047328948975, -0.2211792767047882, 0.2365817278623581, 0.06730296462774277, 1.5819882154464722, 0.43830353021621704, -1.6251499652862549, -0.32383307814598083, -1.3559340238571167, -1.0600767135620117, 0.6171680092811584, -0.177061066031456, 1.050520420074463, -0.016710931435227394, -0.9887387752532959, -1.4455009698867798, -1.2871583700180054, -0.9291995763778687, 2.415234327316284, -0.6762409210205078, 0.1528642177581787, -1.3720033168792725, 0.287070095539093, 0.22257430851459503, 0.6203895211219788, -1.4392844438552856, -0.32478612661361694, -1.8299914598464966, 1.0987340211868286, 0.010171152651309967, 0.0710967555642128, -2.073943614959717, 1.8922182321548462, -1.7306674718856812, 2.737656354904175, 0.6469122171401978, -0.6767134666442871, -0.5611374378204346, -0.3323565423488617, -0.17779213190078735, -0.12004005908966064, -0.0961155891418457, -2.0737149715423584, 0.11528578400611877, -0.0864991694688797, -1.033447027206421, 0.37970173358917236, -0.29260534048080444, -0.4541396498680115, 1.2906142473220825, 0.6387073397636414, -1.7562974691390991, 0.12929660081863403, 0.051871176809072495, -0.44332975149154663, -0.11241092532873154, -1.6715211868286133, 1.8068830966949463, -0.9806217551231384, 0.25227198004722595, 0.13658136129379272, 1.9462978839874268, -0.4431852698326111, -0.7679176330566406, -1.3239003419876099, -0.20840784907341003, 0.09338510036468506, -0.4069015383720398, 0.15782178938388824, -0.4269219934940338, 0.13779328763484955, 0.4426524341106415, -0.29995280504226685, -2.51621150970459, 0.0022846602369099855, -1.0056689977645874, 0.4380577504634857, -0.771165668964386, 2.1738152503967285, 0.28433528542518616, -0.7022075057029724, 0.671710193157196, 0.06439340859651566, -0.7664111256599426, 0.6466835737228394, 0.14317865669727325, 0.6564945578575134, 1.0703086853027344, -0.8448012471199036, 0.056208036839962006, -0.18711644411087036, -0.7046138644218445, -0.5537586212158203, 0.6991015076637268, 0.8854724168777466, 1.67750084400177, 1.2355012893676758, 0.49549347162246704], [0.2524837553501129, -0.23826226592063904, -0.06550565361976624, 0.49058154225349426, -1.626918077468872, 0.2367902547121048, 0.015934770926833153, -0.6570637226104736, 0.945297122001648, 0.32315078377723694, -0.627415120601654, 0.8930124044418335, 1.5113097429275513, 0.8623868823051453, -0.1194668635725975, -1.1347219944000244, 0.027248764410614967, 0.4033128619194031, 2.425447940826416, -1.1607396602630615, -1.4336650371551514, -1.7648502588272095, -1.930333137512207, 0.648368239402771, -0.39765632152557373, 0.07909911870956421, 1.5534636974334717, 0.5694276094436646, 0.1971854269504547, 0.17214705049991608, 1.3740899562835693, 0.11851800233125687, 0.4710807502269745, 0.684714138507843, -0.8095592260360718, -0.7288190126419067, -0.007757535204291344, 1.5875152349472046, 0.5208949446678162, -0.515214741230011, -0.9673864841461182, 0.4888467490673065, -2.2689595222473145, 0.29358503222465515, 2.1265149116516113, 0.005800062790513039, -0.7212867140769958, -1.2522519826889038, -0.6747007966041565, 1.7763190269470215, 1.5149656534194946, 0.9333489537239075, 0.3728146255016327, 1.1677377223968506, -0.8236262202262878, -0.00500616617500782, 0.28523197770118713, 0.16738393902778625, 1.085313320159912, 0.485662043094635, -1.4662561416625977, 0.5516749620437622, -0.1038651168346405, -0.1098487600684166, 0.16007208824157715, 0.07089924067258835, 0.2357284128665924, -0.6418081521987915, 0.30356940627098083, -0.8259989619255066, 0.760009229183197, 1.1465145349502563, 1.6069507598876953, -0.39942169189453125, 0.3496149182319641, -0.5236749649047852, -1.6210402250289917, 0.7215144038200378, 0.10975655913352966, -1.865721583366394, -0.5523295998573303, -0.7078616619110107, -1.1291435956954956, -0.8163885474205017, 0.529038667678833, -0.8046649694442749, -0.39097073674201965, 1.3492313623428345, -1.1431483030319214, -0.8262571692466736, -0.3766688406467438, 1.2737735509872437, 0.19607728719711304, 1.6013535261154175, -0.4872523546218872, 0.9105564951896667, -1.5228655338287354, 0.2481182962656021, 0.4993892312049866, -0.9970421195030212, 0.14171986281871796, 0.11116795986890793, -1.348188877105713, -0.14731420576572418, 1.461421251296997, 0.5738410353660583, -0.2540540099143982, 1.137833833694458, -0.15686264634132385, -0.34730595350265503, 1.2375036478042603, -1.1901861429214478, -1.6759611368179321, 0.793156087398529, 0.169210284948349, 0.15326109528541565, 0.5037152767181396, -1.4776220321655273, -3.3275671005249023, -1.2076126337051392, -0.8085033893585205, 0.32404625415802, 0.594182550907135, -0.1735095977783203, -2.2528016567230225, 2.4550483226776123, 0.7582440376281738, -1.9873093366622925], [-1.762427806854248, -1.192084789276123, 2.680372017493937e-05, 0.08185455203056335, 0.4405253231525421, -0.6218486428260803, -0.38881218433380127, -0.23362407088279724, 0.7053989171981812, 0.045762334018945694, 0.6728762984275818, 0.16057394444942474, -0.9326840043067932, 0.029815834015607834, 0.7220005393028259, -0.896753191947937, 1.8094960451126099, 0.44700655341148376, -0.532911479473114, -0.6468725204467773, 0.9135497808456421, 0.622460663318634, -1.0613316297531128, -0.23394125699996948, -1.1526737213134766, -0.9062275290489197, -0.7116936445236206, 1.536436915397644, 1.4599475860595703, 0.0037021988537162542, 0.3667736351490021, -0.7076192498207092, -2.0256381034851074, -0.6631011962890625, 0.12467911094427109, 1.5210648775100708, 1.5057399272918701, -1.5079188346862793, 0.19028182327747345, -0.10631892085075378, -0.12034282833337784, 1.0457813739776611, 1.0567573308944702, 2.295271396636963, 1.472930908203125, 1.1173218488693237, 0.46271222829818726, 1.1322299242019653, 1.466863989830017, 0.7845969200134277, 0.3059394955635071, -0.12627016007900238, 0.4636193811893463, 1.8466546535491943, -0.5736827850341797, -0.9788786172866821, -0.3914147913455963, 0.8607218265533447, 0.09751478582620621, -0.19208259880542755, 3.4437873363494873, 1.1278908252716064, 0.03930902108550072, 0.8800099492073059, 0.5570074319839478, -0.4148787260055542, -0.5795914530754089, 1.1640039682388306, -1.3843811750411987, -0.10219504684209824, -0.42171189188957214, -1.009526014328003, 1.2853915691375732, -0.9290066957473755, -0.014569913037121296, 0.27870309352874756, 0.10316066443920135, 0.10552489757537842, 0.19240812957286835, -0.07902427017688751, 0.6053169965744019, 0.9647715091705322, 0.9965614080429077, 0.3939811885356903, 0.3834225535392761, -0.2187509387731552, -0.25608280301094055, -0.714895486831665, -0.650497317314148, 3.2277870178222656, -0.4562032222747803, 0.5796638131141663, -0.7643862366676331, 0.8882933259010315, 1.2273616790771484, 0.22141921520233154, -0.6918537020683289, 0.14939424395561218, -0.3970486521720886, 2.4006145000457764, 0.8602530360221863, 0.25269827246665955, -1.4127848148345947, 0.04204656183719635, -1.1816637516021729, -1.2695828676223755, -0.6655547618865967, -0.0006450089858844876, 0.03934358060359955, -0.2959049642086029, -1.168488621711731, 0.42832425236701965, 1.0277135372161865, 1.3598154783248901, -0.7983847856521606, -0.3165707588195801, -1.0366623401641846, 0.5360389947891235, 2.293604850769043, -0.08188894391059875, 1.3529095649719238, -0.28831005096435547, 0.609937846660614, 0.873595118522644, 0.2663496136665344, 1.218404769897461, -2.6864240169525146, -0.6575366854667664], [1.0206507444381714, -0.8343442678451538, -0.4666362702846527, 0.4882301688194275, -0.44258394837379456, 0.8734792470932007, 0.3610915243625641, -1.1691778898239136, -0.18611933290958405, -2.5632262229919434, 0.35799118876457214, 0.8204365968704224, 0.5026938915252686, 1.6731271743774414, 0.7641749978065491, -2.638198137283325, -0.044152501970529556, 2.0473201274871826, -0.8326857089996338, -1.1608924865722656, 0.74752277135849, -0.40298110246658325, -0.03814222291111946, 0.5634633302688599, -1.2221671342849731, 0.6550551056861877, 0.632561206817627, 1.0095006227493286, -1.1251857280731201, 0.1289198249578476, -0.9111809730529785, -0.5687307715415955, 0.8697311282157898, -1.3578630685806274, 1.0964491367340088, 1.3075686693191528, 2.024728536605835, -1.1862549781799316, -0.32614341378211975, -0.7549206614494324, -1.1325595378875732, -0.9754641056060791, -1.0027198791503906, -0.6523907780647278, -0.41934388875961304, -0.44766128063201904, -0.8346887826919556, 0.9367237091064453, -0.493758887052536, 0.0852983295917511, 0.979150652885437, 0.4350954592227936, 1.2480227947235107, 1.104533076286316, 0.7836874723434448, -0.7037001252174377, -1.5103988647460938, 0.329257607460022, 0.030684098601341248, 1.0725629329681396, 0.45518332719802856, -0.45810455083847046, -1.654098629951477, -0.4758945107460022, -0.05210462957620621, 0.2949216663837433, 1.6741660833358765, -1.5282270908355713, -0.33605527877807617, -0.13097412884235382, 0.7931026816368103, 0.6596057415008545, 0.6879376173019409, -0.4393327832221985, -0.6718267202377319, 0.9786757826805115, -1.167933464050293, -0.45763495564460754, 0.7385006546974182, -1.4966249465942383, 0.8805961608886719, -0.6326669454574585, 0.2196875959634781, 0.2392529845237732, -1.2274824380874634, 0.9023923277854919, -0.6811454892158508, 0.685851514339447, 1.5881257057189941, 0.44929614663124084, 0.9465717077255249, 0.8802775740623474, -0.6906211972236633, -0.06276340037584305, -1.303790807723999, 0.3775325417518616, -0.857417643070221, 0.14944539964199066, 0.6304967999458313, 0.9424740076065063, 0.5905694365501404, -1.317159652709961, 2.275902509689331, 0.4414190351963043, 0.1931839883327484, 0.45484867691993713, 1.5725723505020142, 0.36901381611824036, 0.07853888720273972, 0.42637619376182556, 0.12946492433547974, 2.079148054122925, -0.9751450419425964, -1.3328930139541626, -0.3627348244190216, -0.41446781158447266, 0.31569623947143555, -1.0395256280899048, 0.022124020382761955, -0.4895446300506592, -0.3673481345176697, 1.1884517669677734, -0.6716864705085754, -0.28715500235557556, -1.7839421033859253, 1.167210340499878, 0.7336692810058594, 1.7037303447723389], [-1.265405535697937, 0.4287044405937195, 0.5604130029678345, -0.821962833404541, -0.24104250967502594, -1.7742117643356323, -1.0911973714828491, -0.07335972040891647, 1.500464677810669, -0.3861238956451416, 0.04791128635406494, 0.7053163051605225, 1.077221393585205, 1.1234800815582275, 0.3756142258644104, 1.8495261669158936, -0.8120260834693909, 2.543185234069824, -0.3807889223098755, -0.06625350564718246, 0.11837580054998398, -1.8462001085281372, 0.804619312286377, -0.12306543439626694, -0.07095279544591904, -1.2465403079986572, 1.6177388429641724, 0.19424374401569366, -1.1129153966903687, -1.580125331878662, -0.306155800819397, 0.028411246836185455, 0.6287596225738525, 0.5647454857826233, 0.5532609224319458, 0.395906001329422, -2.0329842567443848, -0.8322077989578247, -0.23090910911560059, -0.7853487730026245, 0.9905074834823608, 1.551626205444336, -0.8896405696868896, 1.5179970264434814, -0.5325135588645935, 0.38833069801330566, -0.06654204428195953, 1.8075991868972778, 0.5602331161499023, 0.25776147842407227, 0.5073128342628479, 0.9547812342643738, -1.246209740638733, 0.5849860906600952, 0.29306501150131226, -0.6127350330352783, 1.3735069036483765, 0.6274141669273376, -0.29508116841316223, -0.5495039820671082, -1.431458830833435, 0.5651988387107849, 0.5134323239326477, 0.9625048041343689, -0.11850842088460922, -0.337457537651062, -1.352662205696106, 0.27416664361953735, -0.27822884917259216, 1.6311184167861938, 1.1406439542770386, 0.5000449419021606, -0.22881042957305908, -2.134340286254883, 0.46670976281166077, -0.22449640929698944, 1.556934118270874, -1.0140855312347412, 0.5512295365333557, -1.2686387300491333, -1.8818572759628296, -0.032593365758657455, -0.806751549243927, -0.3697383403778076, -1.2275464534759521, -1.5540648698806763, 1.1813737154006958, 0.47123172879219055, -1.2545477151870728, -0.21646782755851746, -1.2298582792282104, 1.4415024518966675, -0.6489186882972717, 1.18610680103302, -0.6989073157310486, 0.4432486593723297, 0.02136640064418316, 0.9166767001152039, 1.186211109161377, 0.24277834594249725, 1.2693145275115967, 0.38654008507728577, 1.2219350337982178, 0.2533469498157501, 0.8169081807136536, -0.12537731230258942, 0.4839057922363281, -0.2759853005409241, -0.2409900575876236, -2.3239054679870605, -0.8835363984107971, -0.07819323986768723, 1.6608608961105347, -2.734461784362793, -0.4792209565639496, 0.029176272451877594, -0.03738196939229965, 0.4505784809589386, -0.7571143507957458, -1.412224292755127, 0.22433936595916748, 0.46983712911605835, -0.8353843092918396, 0.7389252781867981, -0.2668114900588989, 0.5584309697151184, -0.8904204368591309, -0.5385475754737854], [-0.9311976432800293, -0.010653261095285416, -0.7570925951004028, 0.5785310864448547, 0.0225654449313879, 0.5517839193344116, 0.7239413857460022, 0.2014581561088562, -0.23061928153038025, -0.8600326776504517, 0.05015813186764717, 1.3539464473724365, -0.48082321882247925, 1.547558307647705, -0.180273175239563, -0.4414171278476715, -0.10191989690065384, -0.4630071520805359, 0.02270621806383133, -0.8941230773925781, -0.7964659333229065, -0.12029992789030075, 0.9640699625015259, -3.129502296447754, -2.0964877605438232, -1.4350111484527588, 1.3302204608917236, -0.07360175251960754, -0.7852350473403931, 0.789596676826477, 0.26986798644065857, -0.07664252072572708, -0.4856586158275604, -0.5590471029281616, 0.4561501741409302, 1.1876304149627686, -0.40676596760749817, -0.676462709903717, 0.5708749890327454, 1.646284580230713, -0.26596599817276, 0.6685656309127808, 0.00545937055721879, -0.2975444197654724, 0.08794207870960236, 0.02597324550151825, 0.13802272081375122, -0.5655051469802856, -1.1909422874450684, -1.2261799573898315, 0.9797372817993164, -0.2741956412792206, 0.746959924697876, 1.3416534662246704, -0.8794901371002197, 1.0840338468551636, 1.2554315328598022, 0.3000902831554413, 0.15957069396972656, -0.47705498337745667, 1.439723253250122, 1.4036661386489868, -1.2679933309555054, 1.1420494318008423, 0.12360832095146179, -0.47280988097190857, 0.25068846344947815, 0.7974170446395874, 1.0482277870178223, -0.7332401871681213, 1.3992135524749756, 0.46910253167152405, 0.6881349086761475, 1.1486518383026123, 0.560928225517273, -0.6919224262237549, -0.8997833728790283, -2.008913993835449, -1.7372008562088013, -0.6948908567428589, -0.34068602323532104, 1.3814828395843506, 1.2891894578933716, 0.6774585843086243, 0.03963441401720047, 0.9536728858947754, 0.6818540692329407, 0.618452787399292, -0.6504807472229004, -0.5915144085884094, 1.2516415119171143, 0.7686544060707092, 0.18530704081058502, -0.21571780741214752, 0.23948846757411957, 0.9235127568244934, -0.8635061979293823, -0.4678865671157837, 0.39695659279823303, 0.5465933680534363, -0.8592319488525391, 0.8911420702934265, 0.055219948291778564, -2.370953321456909, -0.011612463742494583, 2.2731850147247314, 1.302925705909729, 0.6660824418067932, 0.6706814765930176, -1.0900721549987793, -0.07159296423196793, -0.9887717366218567, -0.6360064148902893, -0.12304026633501053, 0.5730677247047424, 0.41924944519996643, 0.6342911720275879, 0.44500014185905457, 0.6804412603378296, 0.4250911474227905, -2.0562899112701416, 0.5020449161529541, -1.163745641708374, 0.27106478810310364, -1.1367087364196777, -1.2306692600250244, -1.120657205581665, -0.7509300112724304]]], [[[1.6608656644821167, -0.9929745197296143, -1.4664926528930664, 1.5593538284301758, 0.323598176240921, 0.757665753364563, 0.9351444244384766, -0.10911166667938232, 1.2762407064437866, 0.3569048047065735, -0.02128351479768753, 0.7363632321357727, 2.1075379848480225, -0.2432316541671753, -1.310320258140564, 1.2507199048995972, 0.9651563167572021, -0.9707432389259338, 1.1378215551376343, 0.12056328356266022, 0.46392884850502014, 1.411973237991333, 0.004591512493789196, 1.731569766998291, 0.4194035530090332, -0.46256008744239807, -1.9245142936706543, -1.4973946809768677, -0.3957246243953705, 1.9847126007080078, -1.3726612329483032, 0.9682257175445557, -0.15939831733703613, -1.6612317562103271, -0.12155012786388397, 0.13128015398979187, -0.8224217295646667, -0.5087643265724182, 1.2461973428726196, -2.312772512435913, -0.2083948850631714, -2.531740188598633, -2.150578737258911, -1.045764446258545, -0.004839468747377396, 1.1545219421386719, -0.5290187001228333, 0.742729902267456, 2.11346435546875, 0.6960972547531128, -0.4127572476863861, 0.44406867027282715, 0.031002739444375038, 0.011607306078076363, 1.1542807817459106, -1.4386756420135498, -1.154376745223999, 0.8245162963867188, 0.9304804801940918, 0.6826049089431763, -1.8081326484680176, -0.5775821208953857, -0.025307707488536835, -0.08238117396831512, -1.0728086233139038, 1.7500219345092773, 0.34211933612823486, -0.02345825545489788, -0.5187267065048218, 1.248716115951538, -1.5216227769851685, 0.19622504711151123, -0.2808699607849121, 1.126732349395752, 0.23175044357776642, 1.1441781520843506, 1.6685329675674438, 0.3177569508552551, -1.2700101137161255, -0.3545156717300415, -1.1634658575057983, -0.7355013489723206, 0.6993885636329651, -0.07230640947818756, 0.9456909894943237, -0.7560347318649292, -1.2189031839370728, 1.2628540992736816, -1.2726542949676514, 1.335913062095642, -1.6124646663665771, 1.6375329494476318, 2.836073160171509, 0.24912308156490326, -0.9724614024162292, -0.9617464542388916, -1.4984928369522095, -1.7763915061950684, -1.5103925466537476, -0.044464435428380966, 0.23253333568572998, -0.33025407791137695, -0.8035099506378174, -0.3087083399295807, 0.2541203200817108, 1.1691255569458008, -0.8786315321922302, 1.3742446899414062, 0.5533982515335083, -2.154409170150757, -0.45522186160087585, -0.3390860855579376, 1.1054835319519043, -0.8274555206298828, 0.16107313334941864, -0.32272636890411377, -1.051092505455017, 0.1127023696899414, -0.723507285118103, -1.2409087419509888, 0.8511985540390015, 0.5064923167228699, -0.35715246200561523, 0.1395287662744522, 0.7020984292030334, 1.739052176475525, -1.165784239768982, 0.726015567779541], [-1.9964401721954346, -2.102613687515259, -0.5806875228881836, -0.9263618588447571, -0.6610506772994995, 0.05767769739031792, 0.06491675227880478, 0.5999616384506226, 0.032972563058137894, 0.7766328454017639, 1.4897191524505615, -0.1506425142288208, -0.694260835647583, -0.7318558096885681, -0.882606565952301, -0.003549721324816346, 1.211919903755188, 0.4418219029903412, 0.2794354557991028, -0.6136555671691895, 1.9241523742675781, -0.20130415260791779, 1.3728467226028442, 1.074897050857544, -0.830913245677948, 0.8463963866233826, 0.5345578193664551, 0.3633800745010376, 0.87093186378479, 1.1358089447021484, 0.054645899683237076, 0.03333840146660805, -1.3271987438201904, -0.7948142290115356, 0.29215267300605774, -0.6557037234306335, 0.21437177062034607, 1.3473055362701416, 1.0751841068267822, -1.2700915336608887, -0.6293333172798157, 0.7428802847862244, -0.32127645611763, 0.4263059198856354, -1.167319416999817, 1.0324734449386597, 1.1434857845306396, 1.0607925653457642, -0.1975180208683014, -0.7319194078445435, -0.03337676450610161, -2.373286247253418, 1.2610384225845337, 0.8754037022590637, -0.030399002134799957, 2.107816696166992, 0.8136743307113647, 1.0629267692565918, -0.0629933625459671, 0.3486992418766022, 1.6197625398635864, 1.3469158411026, 0.38682788610458374, 0.2673190236091614, -1.3052281141281128, -0.06004280596971512, 1.5912210941314697, 1.757978081703186, -1.448807954788208, -0.013991937041282654, 0.14758343994617462, -0.790715754032135, 0.8143147826194763, 0.5694617629051208, 1.1343491077423096, 0.15684626996517181, -0.3711917996406555, 1.008001446723938, 0.6580455303192139, -0.407978892326355, -0.7474397420883179, 0.05719431862235069, -1.072521686553955, 0.3713490664958954, -0.6068120002746582, 0.005547694396227598, -1.912523627281189, -0.11666914075613022, -0.7520172595977783, -0.6137784123420715, 0.5423376560211182, 1.022599697113037, 1.0444300174713135, -0.1271769255399704, -0.028162222355604172, -0.657602846622467, -0.16233950853347778, 0.6026577353477478, 0.8383723497390747, 1.4840139150619507, -0.32044124603271484, -0.5197470188140869, 0.21108444035053253, -0.22925087809562683, -0.5789276957511902, -1.4468170404434204, -1.456221342086792, 0.9396435022354126, -0.45113199949264526, -1.0969547033309937, -2.893402576446533, 0.2612963020801544, 1.5098720788955688, -0.3452506363391876, -0.8752186298370361, -0.09370003640651703, -0.9830496311187744, -1.3803547620773315, -1.0791995525360107, 0.03987952694296837, -1.5916653871536255, 0.5268546938896179, 0.2856534719467163, 0.2835196554660797, -0.28825676441192627, -0.8319708108901978, -0.4969594478607178, 1.2685812711715698], [-0.21425947546958923, 0.04743330925703049, 0.04183292016386986, -0.9521557688713074, -0.2549390196800232, 0.2640702724456787, -0.24515432119369507, 2.7085301876068115, -0.7172317504882812, 0.0863603875041008, 1.5584471225738525, 1.0964680910110474, -1.1289174556732178, 0.26004815101623535, 0.07578536868095398, 1.727597951889038, 0.3986194431781769, -0.23885244131088257, 1.1143220663070679, -0.743988573551178, 0.14447292685508728, -0.9306420087814331, -1.1180222034454346, -0.9510906338691711, -0.4014800488948822, -1.629534125328064, -1.4128077030181885, 1.06931734085083, 0.6250808238983154, 1.2099295854568481, 0.6858812570571899, 1.1849840879440308, 1.6503355503082275, -1.3740466833114624, -1.3238645792007446, 0.7332442402839661, 0.5069093704223633, -0.12420342862606049, 0.3489607274532318, -0.6666639447212219, -1.4242033958435059, -0.6643447279930115, -0.2374565303325653, 1.1377874612808228, -1.1111996173858643, -0.8245950937271118, -0.577235758304596, -0.8728966116905212, -0.18584981560707092, 1.14798903465271, -0.007883480750024319, 0.3873642086982727, 0.2364063858985901, 1.3551119565963745, 1.2356783151626587, -1.4658328294754028, 0.4515629708766937, -0.2796967327594757, 0.37501561641693115, 1.0917223691940308, -0.18115931749343872, -1.1488875150680542, 1.2097023725509644, -0.08742481470108032, 0.5099914073944092, -1.3383513689041138, -1.3962405920028687, -0.4668503999710083, 0.07751885056495667, 1.706024169921875, -1.2570642232894897, 1.5139487981796265, -0.4989013075828552, 0.5843784213066101, -0.006072235759347677, 0.7935488820075989, -0.9904910922050476, 0.43041834235191345, 1.1224088668823242, -1.8753197193145752, -0.8261761665344238, -0.8801229596138, -0.8069900870323181, 0.24560290575027466, -0.1020481139421463, -0.23527707159519196, 1.362221598625183, -1.8614193201065063, 1.0188952684402466, -0.38077765703201294, 1.3751821517944336, 0.34871289134025574, -1.1513272523880005, 1.961682677268982, -0.4960632026195526, 2.2546563148498535, -0.024079402908682823, 1.2682411670684814, -0.16729941964149475, -0.4267549514770508, 0.7643177509307861, -0.18017350137233734, 0.15919093787670135, -0.7109624743461609, -0.9690541625022888, 0.4368363916873932, 0.22078144550323486, 0.37795573472976685, -0.4079159200191498, -0.8279685378074646, -2.120908260345459, 0.28649064898490906, 0.26290780305862427, -0.25249752402305603, -0.13653624057769775, -0.5440382957458496, -0.30464744567871094, -0.2337121218442917, -0.025857893750071526, -2.1334593296051025, -0.11507615447044373, -1.122147560119629, 0.5497798323631287, -0.196129709482193, 0.6313197016716003, -1.7358348369598389, -1.493788480758667, -1.6267731189727783], [-0.5505497455596924, -0.19811129570007324, -1.2503803968429565, 1.970686912536621, -0.34703347086906433, -0.7005438804626465, 0.7811746001243591, -2.078188180923462, 0.917831301689148, -1.4009722471237183, -0.6386042833328247, -2.046955108642578, 1.6922107934951782, -0.8548984527587891, 0.7363781332969666, -2.5749285221099854, 0.3938335180282593, 0.8748101592063904, -1.5322303771972656, 2.2999932765960693, -0.2676171660423279, 1.069042444229126, 1.2298487424850464, 0.353139728307724, -1.3670815229415894, -0.43541523814201355, 1.2749042510986328, -0.7197731733322144, 2.4390084743499756, -1.489762544631958, -0.3429661691188812, 1.6275484561920166, -1.1054552793502808, 1.0096174478530884, 0.3875340521335602, 0.8096805810928345, -0.5026800036430359, 1.4568090438842773, 0.9503998756408691, 1.6294820308685303, -0.2234473079442978, 0.032149266451597214, 0.013810363598167896, -0.4363330900669098, -0.38910236954689026, 0.16126656532287598, -0.011536865495145321, -0.23545575141906738, 0.48381486535072327, -0.37926918268203735, -0.150893896818161, 2.2877988815307617, 0.28481703996658325, -0.18002161383628845, 0.9319416284561157, -1.245549201965332, 0.5322572588920593, -0.1984012871980667, -1.2136963605880737, -0.28181931376457214, 0.7899413108825684, 0.0013946564868092537, -0.5923289656639099, -0.4014548361301422, 0.5223828554153442, 0.7784428596496582, 0.8156693577766418, -0.9041280150413513, 0.6158870458602905, 0.10728075355291367, -0.7560591697692871, 1.2041072845458984, -0.5837329626083374, -1.0430355072021484, -0.9302859306335449, 0.08848828822374344, -1.586658000946045, 1.9382225275039673, -0.7304549813270569, 0.13548687100410461, -0.6780520081520081, 1.5300674438476562, 1.5028893947601318, -1.1724594831466675, -0.23621906340122223, 1.6747589111328125, -1.8213982582092285, 1.0336859226226807, -0.9314408898353577, -1.0511693954467773, 1.2060340642929077, -0.26308178901672363, -1.665157437324524, -0.7799237370491028, 0.2889349162578583, 0.32908061146736145, 0.4736088812351227, 1.6749053001403809, 1.3448078632354736, 0.44269463419914246, 0.27756166458129883, 0.19386717677116394, -0.36559128761291504, 1.1349494457244873, -0.1106841117143631, 0.5822001695632935, 0.36714082956314087, 0.8001604080200195, -0.7908287048339844, -0.7062769532203674, -0.5635712146759033, -0.5165150761604309, 0.7665253281593323, 0.03231004253029823, -0.9914653897285461, 0.4724433720111847, -1.389217734336853, -0.3901178538799286, -0.0513521246612072, 0.686750054359436, 0.1727239340543747, 0.1268075555562973, 0.5902796387672424, 1.138857126235962, -0.050068601965904236, 0.7197964787483215, 0.6090124249458313, -1.0207374095916748], [-0.42329615354537964, 0.374260812997818, -0.5466390252113342, 1.6564087867736816, -0.16264067590236664, -0.4069341719150543, -0.2781166434288025, 2.0701375007629395, 1.924803614616394, 0.2352054864168167, -2.1134562492370605, 0.7358654141426086, 0.0837399885058403, -0.030112670734524727, 0.5849205255508423, -0.8661463856697083, -0.48526743054389954, 0.3111029267311096, 1.3361244201660156, 0.3293566405773163, 1.0437642335891724, 0.5073984861373901, -0.14203263819217682, -0.17382246255874634, 0.519723117351532, -0.013471085578203201, -0.7186289429664612, -0.14200715720653534, -0.02063458226621151, -0.2088194191455841, 0.417955607175827, -0.9985076189041138, 0.8880451917648315, 1.226322054862976, 0.2782800793647766, -1.6566035747528076, -0.43031179904937744, -0.9696776866912842, -0.36367061734199524, -0.3774700462818146, 0.9819931387901306, 1.2186641693115234, 0.8736507892608643, -0.5311530828475952, -0.2280960977077484, 0.5328569412231445, 0.4413386583328247, -1.5528638362884521, -0.20227952301502228, 0.0795259028673172, -0.46790623664855957, 2.292637348175049, 1.6032639741897583, -2.660183906555176, -1.1051865816116333, 1.821772813796997, -0.8442822098731995, 0.2142883539199829, -0.40737995505332947, -1.2910830974578857, 0.1712268739938736, 0.7816287279129028, 0.2521945536136627, -0.956420361995697, -0.8584909439086914, 0.7925528883934021, 1.240185022354126, -0.8250104188919067, -0.8837298154830933, 0.41938695311546326, -0.09423372894525528, -0.30183565616607666, -0.9396365880966187, 0.915492594242096, -0.9605063796043396, 0.22424915432929993, 1.0694154500961304, 1.627454400062561, 0.16402284801006317, -1.4762765169143677, 0.9433311820030212, 0.7251377701759338, -1.1928293704986572, 0.5103395581245422, -1.8415634632110596, 0.03808846324682236, 0.33540695905685425, -1.95828115940094, -0.08071566373109818, 0.5632948875427246, 0.025779830291867256, 1.2021067142486572, -1.9634782075881958, 0.10646059364080429, 0.18975059688091278, 0.7955427765846252, 1.0787428617477417, -0.9484449028968811, -0.6604211330413818, 0.005972529761493206, -0.024437561631202698, 1.003495216369629, -0.4940679669380188, -0.3500893712043762, 0.6085608005523682, -0.817243754863739, 0.310242235660553, 1.8204902410507202, 1.2530757188796997, -0.904984176158905, 1.0266780853271484, -0.5011579990386963, -1.3449394702911377, 0.4492053687572479, 0.4077381491661072, 0.06738133728504181, 0.1362621933221817, 0.3513287603855133, 0.740542471408844, 0.28688183426856995, -1.1841801404953003, -1.4931175708770752, 0.05121924728155136, 0.03158798813819885, -1.1375696659088135, 1.1725728511810303, -0.15155211091041565, 1.4196239709854126], [0.21325312554836273, -0.2265487164258957, -0.8951221704483032, 0.18149760365486145, 2.115088939666748, 1.0226398706436157, 1.7006702423095703, -0.8427842855453491, -0.37697726488113403, 0.7660285830497742, 1.0280184745788574, 1.946890115737915, -0.8587013483047485, -0.638937771320343, 1.2190477848052979, 0.7948975563049316, -0.9943915009498596, -1.0581440925598145, 0.16633601486682892, 1.2668043375015259, 0.3515658378601074, -0.2236211746931076, 0.710072934627533, 1.2356159687042236, 0.027515588328242302, -0.7804701328277588, -0.046119946986436844, 0.51133131980896, -0.8205925822257996, 0.12152097374200821, 0.08669048547744751, 0.24135549366474152, -0.3906519412994385, -0.05820588022470474, -0.562440812587738, 1.4010130167007446, -0.46702954173088074, 1.321550726890564, -0.44516682624816895, 0.14952951669692993, -1.1781460046768188, 0.6660581231117249, -0.4738917350769043, -0.4316524565219879, 1.2451163530349731, 1.4096813201904297, 0.7838598489761353, -0.9156242609024048, -0.989404022693634, -0.24843628704547882, 0.13946989178657532, 0.725165605545044, -0.22134317457675934, -0.8700234889984131, 0.15591329336166382, 0.8569458723068237, -0.2529126703739166, 0.8121943473815918, 0.9183804392814636, 0.44604629278182983, -0.3509660065174103, 0.4238583445549011, 0.44086915254592896, -0.5918261408805847, 2.142254114151001, -1.5980021953582764, -0.32266470789909363, 0.21759121119976044, 0.3654792904853821, -1.1719856262207031, 0.025933794677257538, -0.2854575216770172, 0.2629894018173218, 1.7478817701339722, 1.4255073070526123, 0.4472638666629791, -1.2797534465789795, -2.346665620803833, 0.18827883899211884, 0.019658681005239487, -1.151668906211853, 0.28725236654281616, -1.3149542808532715, 0.1391899585723877, -1.5626435279846191, -0.5551157593727112, 1.4744371175765991, 0.41547614336013794, -0.2172292321920395, -0.3576202392578125, -1.2623823881149292, 0.8107835650444031, 0.6202383041381836, -0.4756668210029602, -0.09669719636440277, 1.0498249530792236, 0.3196662962436676, 0.6893523335456848, -0.5378246903419495, -0.8362372517585754, 0.053116798400878906, -0.7685422897338867, -0.3959483504295349, -0.740850567817688, 0.34626343846321106, -0.3190392553806305, -1.7655636072158813, 2.0745129585266113, 0.023957904428243637, -0.4008105993270874, 0.4034699499607086, -1.6238148212432861, -0.07468023151159286, 0.8293211460113525, 0.6020545363426208, -0.5018529891967773, 1.4828306436538696, -0.46727582812309265, -1.0306295156478882, -1.383837103843689, 1.0790728330612183, 1.3638397455215454, 1.4226313829421997, -0.6180285215377808, 1.6262081861495972, -0.6931120157241821, -0.536689043045044, -0.21852600574493408], [-1.2084672451019287, 1.0541749000549316, 1.3651665449142456, -2.0696356296539307, 0.5756976008415222, -0.5177742838859558, 0.6408157348632812, -0.7908511161804199, 0.8447362780570984, -0.7069944143295288, -0.7368674278259277, 0.029601242393255234, 0.6090594530105591, 0.30679431557655334, 1.6609987020492554, 1.598730206489563, 0.18595272302627563, -0.0932072177529335, 1.5814826488494873, -1.2140213251113892, -1.012337327003479, 0.3551267087459564, -0.9317677617073059, -0.423644095659256, -0.9145358204841614, 0.15735264122486115, 0.2002507746219635, -2.275240898132324, -0.14549387991428375, 2.2651584148406982, -1.704550862312317, 0.8804214000701904, 0.4839012026786804, -1.0924162864685059, -0.8971792459487915, -1.2025526762008667, 1.1738667488098145, 0.37889304757118225, -0.21303488314151764, 0.48240959644317627, 0.34642133116722107, -0.7262177467346191, -0.9334153532981873, -0.7177596688270569, 0.2671647071838379, 0.7389867901802063, 1.073197364807129, -1.408681035041809, -1.0188122987747192, -0.4597577154636383, 0.7741724848747253, -0.3652275800704956, 0.07365076243877411, -1.4451203346252441, -0.9994686245918274, 0.0992342159152031, 3.4894139766693115, 0.19461855292320251, -0.8706644773483276, -0.8052988052368164, 0.9604918360710144, -1.338229775428772, -0.02010454051196575, -0.9928267002105713, 2.1243324279785156, -0.005385149735957384, -0.9802179336547852, -1.9932650327682495, 2.694922924041748, -0.2701070308685303, 2.4632620811462402, -0.4778269827365875, -0.15151186287403107, 0.293975293636322, 0.011296534910798073, 1.9517401456832886, -0.6230517625808716, 0.25407490134239197, 0.11877437680959702, -0.10430622845888138, 1.5134817361831665, -0.8501845002174377, 0.22943614423274994, -1.4847517013549805, -0.2845275104045868, -0.5925238728523254, 2.1018893718719482, -0.4658720791339874, -0.09868551790714264, -0.11169172078371048, -1.6586109399795532, -0.9076537489891052, -0.1494247168302536, 0.30224236845970154, 2.732966423034668, -0.13055285811424255, 1.6305445432662964, 0.1982666403055191, 0.23139254748821259, 0.8742870092391968, 1.4441559314727783, 1.2738527059555054, -1.0766146183013916, -1.0735377073287964, -0.29810482263565063, 0.008125335909426212, 1.3808653354644775, -1.6902947425842285, -1.5253868103027344, -1.9529902935028076, 0.7053653597831726, -0.4959488809108734, -0.10225196927785873, 0.5127333402633667, 0.18421940505504608, 1.2294895648956299, 0.6113128662109375, 1.5897150039672852, 0.0415123887360096, -0.21238583326339722, -0.6175408363342285, -0.08225081115961075, -1.1115788221359253, 0.8631501793861389, -0.11252758651971817, -0.6462022662162781, 1.2721353769302368, 1.6543360948562622], [1.022757649421692, -2.2485408782958984, -0.061324797570705414, 0.04033883661031723, -0.6012592315673828, -0.34097936749458313, -1.7580856084823608, -2.350588083267212, 0.3894687294960022, -0.6289887428283691, -0.12479566782712936, 2.2543222904205322, -0.30246442556381226, 1.0687662363052368, -0.152569979429245, -0.18886114656925201, -0.23376010358333588, -0.13251014053821564, 0.398053914308548, -0.4570693075656891, 0.6686216592788696, 1.3027976751327515, -0.28847798705101013, -1.5340723991394043, 0.6923918128013611, 0.2687605917453766, 0.3643264174461365, 0.8648049831390381, -0.39405906200408936, 1.0306800603866577, -0.8014458417892456, 0.0013223804999142885, -0.19449687004089355, -1.1380184888839722, -0.01857464201748371, 0.05714303255081177, -0.6422630548477173, -0.25453421473503113, -1.158976435661316, -1.0445538759231567, -0.02118220552802086, -2.0568480491638184, 0.7654914259910583, 0.34307870268821716, -0.21676260232925415, -0.22810685634613037, 0.7639815807342529, -1.244537353515625, -0.08552223443984985, 0.11674462258815765, -0.05822204425930977, 0.5022276639938354, -1.2048274278640747, -0.24159491062164307, 2.6175942420959473, -0.9872788190841675, 1.0551433563232422, 0.40472373366355896, 0.7994751334190369, 0.7173520922660828, -0.10824253410100937, 0.7081350088119507, -0.7638024091720581, -0.5341587662696838, 1.6476397514343262, -1.0677096843719482, -0.9935702681541443, 0.37024787068367004, 1.3605356216430664, 1.771144151687622, 0.956719160079956, 0.46976929903030396, 1.0967836380004883, 1.2428722381591797, 0.2160635143518448, 1.5374175310134888, 1.0644471645355225, 0.0878627598285675, 0.6563916802406311, -0.501787543296814, 0.24789230525493622, 1.4956002235412598, -1.8932912349700928, -0.6238957643508911, -0.2061724066734314, -0.585587203502655, -0.9033013582229614, -0.46489763259887695, -1.9105452299118042, 0.17764565348625183, -1.5147494077682495, 0.9029554724693298, -0.5225750207901001, 0.9421055316925049, -0.40683698654174805, -0.04963771253824234, -1.1446563005447388, 2.446859836578369, -1.2747926712036133, -1.2485313415527344, 1.1595114469528198, 2.1812584400177, -0.4541035592556, -0.4300854206085205, 0.46964746713638306, -0.7943868637084961, 1.7907469272613525, 1.327099323272705, -0.4583558142185211, 1.6002519130706787, -1.9904661178588867, 0.3867639899253845, 0.01669127121567726, 0.4112633168697357, 0.49095648527145386, -0.7039434909820557, 1.201362133026123, -0.0585351288318634, -0.3655329644680023, 1.0881221294403076, -0.6653012037277222, 0.7524541616439819, 1.299232006072998, 0.4018813967704773, -0.085985466837883, 0.4014609456062317, -2.286958932876587, 0.8663009405136108], [0.0994717925786972, 0.12991583347320557, 1.6925324201583862, 0.42592954635620117, -0.11450473219156265, -0.7598435282707214, -3.028596878051758, -0.2809191048145294, -1.7216169834136963, -0.6715723872184753, -0.35963600873947144, 0.5090265274047852, 0.3344125747680664, 0.9019045829772949, 0.6557528972625732, -0.8967524170875549, -0.5098535418510437, -0.25196731090545654, -0.35193386673927307, -1.1453872919082642, -1.0942983627319336, 0.9641091823577881, -0.738054096698761, 0.5357745289802551, 0.2962540090084076, 0.5631028413772583, -1.2623950242996216, -1.4992376565933228, 0.4722885191440582, 0.2638764977455139, 2.2236075401306152, 0.1267232596874237, 0.45699572563171387, 0.5051693916320801, -0.7657449245452881, -0.0755622535943985, -2.812758445739746, -1.3845726251602173, 0.7062432169914246, -0.15713322162628174, 1.0564771890640259, 0.011784477159380913, 1.441954493522644, -2.576490879058838, 0.782556414604187, 2.0309364795684814, 0.6901929974555969, 0.6921488046646118, 0.0028782107401639223, -1.1485612392425537, 0.652212917804718, 1.1906648874282837, 0.4352533519268036, -0.2048037201166153, -1.5772087574005127, -0.6890736818313599, -1.032578945159912, 0.27287763357162476, -1.4621617794036865, -0.4606151580810547, 1.254097580909729, 1.2916845083236694, 0.8921424746513367, 1.4753766059875488, 1.8143049478530884, 0.8671904802322388, 1.3864188194274902, 0.046202968806028366, 2.436171770095825, -0.9175273776054382, 0.7261912226676941, -0.1676904410123825, 0.6081558465957642, -2.4557042121887207, 1.5569778680801392, -0.8993818759918213, 1.7048577070236206, -0.40154024958610535, 0.41865605115890503, -2.2871320247650146, -0.08426282554864883, 0.4638581871986389, -0.022318346425890923, -2.262563943862915, -1.9619550704956055, 1.8197516202926636, 0.08110889792442322, 0.41396796703338623, 1.3123869895935059, -0.2946809232234955, -0.3299158215522766, -0.39627009630203247, -1.9153637886047363, -0.5903897285461426, 1.3985531330108643, -0.877813994884491, -1.3732329607009888, -0.5431124567985535, 0.6512207388877869, 0.896221399307251, -0.8703998327255249, 0.3366457521915436, -0.3789118230342865, 0.946882963180542, -0.5021702647209167, 1.0013625621795654, -1.0985300540924072, 1.459303617477417, -0.6722028851509094, 1.0064777135849, -0.8265585899353027, 1.2456300258636475, -0.7070431709289551, -0.2623065710067749, -0.818359911441803, -1.2435134649276733, -0.9638063311576843, 0.5855305790901184, 0.8150810599327087, -0.7183953523635864, 0.13251321017742157, -1.068956732749939, 2.061819553375244, 0.23914214968681335, 0.33677157759666443, 0.012062237598001957, -0.6343773603439331, 0.3005934953689575], [0.036507006734609604, 0.16433782875537872, -0.6977784633636475, -0.7931747436523438, -1.4726282358169556, 2.261883020401001, -0.4331367611885071, -0.9883637428283691, -1.1932157278060913, -0.45378100872039795, -1.731978178024292, -0.1342535763978958, -3.4076943397521973, 1.439501166343689, 0.7818018794059753, -1.4371260404586792, 1.449208378791809, -1.8149150609970093, 0.3883627653121948, -0.14656320214271545, 0.7552084922790527, 1.0031286478042603, -0.599719762802124, 1.7827562093734741, -1.0401784181594849, -0.26715293526649475, -0.28521308302879333, 0.5906788110733032, -1.1762909889221191, 0.5704147219657898, -0.8031165599822998, -1.5368846654891968, 0.9240491390228271, 1.73393976688385, -1.6735681295394897, 0.7314565777778625, 1.042913794517517, 1.2604351043701172, -0.5842434167861938, -1.2417984008789062, 1.0283282995224, 0.6391215920448303, 0.45960402488708496, 0.024658391252160072, -0.5590133666992188, 0.7243709564208984, -0.6563981771469116, 0.7045620679855347, 0.1261843740940094, 0.6748930811882019, 0.932131826877594, 0.05026141181588173, -2.4550211429595947, 0.9571559429168701, -0.5447468161582947, -0.2813907861709595, -0.19610550999641418, -0.07282672822475433, -0.009099706076085567, -0.9450862407684326, 0.3599216938018799, -0.1298905611038208, -1.0744119882583618, 0.6925168633460999, 0.39902257919311523, 1.2893232107162476, -0.2478407919406891, -0.3088158369064331, -0.568922758102417, 1.4011948108673096, 1.7944154739379883, -1.6999605894088745, 0.43965640664100647, -1.8042075634002686, 0.6484729647636414, 2.880981683731079, 0.0066817994229495525, -0.06535545736551285, 0.18445958197116852, -0.11925952136516571, 0.06708072870969772, 0.09057022631168365, 0.23190927505493164, 1.527382731437683, 0.1580367088317871, 0.37270402908325195, -0.08177217096090317, 0.2715899348258972, -0.7798125147819519, -0.3056286573410034, 0.23411646485328674, 1.0310914516448975, -1.4879592657089233, -0.47385600209236145, 0.8040971755981445, 0.32424789667129517, -0.8488737344741821, -0.08205536752939224, -0.33796870708465576, -0.18066145479679108, 1.1996922492980957, -1.4324829578399658, -1.3917289972305298, -0.8986368179321289, -0.33320367336273193, -0.5504295825958252, -0.11778708547353745, 0.6037957072257996, -0.6897650361061096, -1.1361833810806274, -0.4052535891532898, -1.2294672727584839, 1.5229562520980835, -0.08462761342525482, 0.29669007658958435, 1.533360481262207, -0.6179018020629883, 0.8838189840316772, -0.2672063708305359, -1.9907851219177246, 0.2842390537261963, -0.2665040194988251, 1.4568812847137451, -0.21332690119743347, -0.4039982259273529, 0.3198189437389374, 2.4455220699310303, 0.9726549983024597], [0.5474442839622498, -1.7027907371520996, 1.2150269746780396, 1.3060178756713867, -0.7378864884376526, -0.8369831442832947, -0.7488533854484558, 0.19480006396770477, 0.6317927837371826, 0.10694040358066559, -0.9166567921638489, -1.6426575183868408, -0.7940881252288818, -1.0712090730667114, 0.06527407467365265, 1.3544505834579468, 2.8072993755340576, -1.3430380821228027, -1.4863622188568115, -1.1886786222457886, -1.3404486179351807, 0.25887301564216614, 1.1745059490203857, 0.4329055845737457, -1.7630231380462646, 0.15831436216831207, 0.32801908254623413, 0.31587761640548706, 0.4973222613334656, -2.0106492042541504, -1.7448753118515015, -0.28803110122680664, -0.14166885614395142, -0.2805633246898651, 0.21892684698104858, -2.4763906002044678, -0.8076825737953186, -0.09511295706033707, -0.15891984105110168, -1.342240571975708, -1.1311206817626953, -0.36684712767601013, -1.2229803800582886, -0.7187286615371704, 0.17186357080936432, -0.6638322472572327, 0.44719114899635315, 0.44680142402648926, -0.9666215181350708, -1.5783871412277222, -0.3274833559989929, -0.5409469604492188, -0.8201186656951904, -1.2373263835906982, -0.6597945094108582, 0.8073635101318359, -0.6419575214385986, -0.3511573374271393, 0.45758387446403503, -0.7134923338890076, -0.553678572177887, -1.2576181888580322, 0.8374671339988708, 0.661791980266571, -0.2757124900817871, -0.6827488541603088, 0.431869238615036, 0.8729431629180908, -1.0428235530853271, -3.300081729888916, 0.5608576536178589, 1.371705412864685, -0.175057053565979, -0.032005660235881805, -0.38340359926223755, 0.47215017676353455, -2.201064348220825, -0.16954554617404938, 1.4752322435379028, -1.5464768409729004, -0.2896258234977722, 0.712757408618927, -0.5952954888343811, -2.1673154830932617, 0.3847498297691345, -0.11629974097013474, 0.36704760789871216, -0.4484100043773651, -0.6489123702049255, 0.25737306475639343, 0.8874857425689697, 0.9845581650733948, -1.3466421365737915, -1.4707897901535034, -1.0573883056640625, 0.5434829592704773, -0.5933610796928406, -1.5719380378723145, -0.18262708187103271, -0.57086181640625, 1.04655921459198, -0.45804035663604736, -0.5520979166030884, 0.08727798610925674, 0.5665701627731323, 1.427794098854065, -0.9210771918296814, -0.6922375559806824, -1.4995795488357544, 0.331237256526947, -0.2503642737865448, 1.2840572595596313, -0.6135779023170471, -1.1807366609573364, 0.8651093244552612, 0.9343641400337219, 0.4739750325679779, -2.3606200218200684, -0.35377439856529236, -0.7380313873291016, 0.18677020072937012, -0.783704400062561, -1.2286995649337769, -0.2729666829109192, 0.5599104166030884, 0.3287827670574188, -0.38865047693252563, -1.1826320886611938], [0.4452802836894989, -0.46280813217163086, -0.9787241816520691, 0.26952338218688965, -1.5372021198272705, -0.14367111027240753, -0.8751639723777771, 1.0765753984451294, -0.929530143737793, -1.8320473432540894, -0.41575533151626587, -0.9144341945648193, 1.3311471939086914, 0.39053088426589966, 0.27020829916000366, 1.6083258390426636, 0.8761253356933594, 0.07751172035932541, -1.4879497289657593, 1.0112404823303223, -0.8227606415748596, -1.7524347305297852, 0.5485247373580933, 0.8905932307243347, 1.0265377759933472, -1.7586607933044434, 0.5915188789367676, 0.05457882210612297, -1.979109287261963, 0.2237618863582611, -2.7430813312530518, 0.25954264402389526, 0.11993300914764404, 1.7344136238098145, 1.0338977575302124, -0.07941362261772156, 0.2848484218120575, -0.908541202545166, 1.5515248775482178, -2.4821064472198486, 0.15056505799293518, -0.9470052123069763, 0.6000623106956482, -0.33494898676872253, -1.0081204175949097, -1.956394910812378, 0.09504258632659912, 0.4140794277191162, 1.3416043519973755, 0.29599928855895996, -0.36764436960220337, -0.4601384997367859, -0.5965626835823059, -0.8546280264854431, -0.018494250252842903, -1.0048059225082397, 0.4343656003475189, -0.28960853815078735, -0.2832358181476593, -0.5819768309593201, -0.04969557747244835, -1.2389237880706787, -0.33316078782081604, -1.4617273807525635, 0.21849007904529572, -1.012403964996338, 1.5685112476348877, -1.5300718545913696, -0.38431715965270996, 0.8776047825813293, 0.42282894253730774, -0.7069449424743652, -0.17936882376670837, -0.3944857716560364, 0.16277751326560974, 1.6055485010147095, 1.438416600227356, 0.33187925815582275, 1.9063475131988525, -0.07589533925056458, -0.5015034079551697, -0.4638225734233856, 3.437899112701416, -0.31263619661331177, -0.23147493600845337, 0.49368250370025635, 1.5658724308013916, 1.1159769296646118, -0.07940517365932465, 1.7310924530029297, 0.06214156746864319, -0.5936986207962036, 0.6688716411590576, 0.736740231513977, 0.05811058729887009, 0.4076194763183594, 0.2651849687099457, -1.412164330482483, -0.5303454399108887, -0.2099870890378952, 0.6250857710838318, 0.7842261791229248, 0.36839988827705383, 0.0855569839477539, -0.5981276035308838, -1.0403608083724976, 0.47742244601249695, 1.1110833883285522, 0.41467148065567017, -0.30566951632499695, -1.2570301294326782, 0.6259958148002625, 0.3680650293827057, 0.512881338596344, 0.2314450740814209, 0.41585060954093933, -0.4211511015892029, -1.2338064908981323, 0.06007340922951698, 0.17601844668388367, 0.46631568670272827, -1.0181950330734253, 0.9890783429145813, -1.437015175819397, -0.07302717864513397, 0.009942862205207348, 1.4695838689804077, -1.1534689664840698], [-0.45119762420654297, 3.3546390533447266, 0.07112854719161987, -0.2508847117424011, 1.2336630821228027, -0.8118817806243896, -0.6285130381584167, -0.3408457636833191, -1.5195140838623047, 0.03048595041036606, -1.5138704776763916, 1.325758457183838, -1.4898879528045654, -1.4845789670944214, 0.3455445468425751, -0.3744581639766693, 0.1545288860797882, 0.08724501729011536, -2.216294765472412, -0.38605034351348877, -0.3396153450012207, -1.256691336631775, 1.0718178749084473, -1.1270076036453247, 0.4298567771911621, -0.10670655220746994, 0.8155755996704102, 0.19331087172031403, -0.7211631536483765, 0.08315111696720123, -0.3350985050201416, 1.3696836233139038, -1.2024472951889038, 0.13967512547969818, 0.5103082656860352, 1.2235392332077026, -0.3279326260089874, 0.041676115244627, 2.309868335723877, 0.595820963382721, -0.35420867800712585, -0.42936861515045166, 1.0943753719329834, -1.989629864692688, -0.24998535215854645, -1.1737020015716553, 1.5606571435928345, -0.386967271566391, -0.12523362040519714, 0.6137654185295105, -0.7264904975891113, 0.21235470473766327, 0.14747540652751923, -0.11889281868934631, 0.4874245524406433, 0.138367161154747, -1.7643378973007202, 0.4183526039123535, 0.5193273425102234, -1.977573275566101, 1.5481553077697754, -0.7226606607437134, -1.2046540975570679, -0.8836844563484192, 0.3806726336479187, 0.3643900156021118, 1.3003016710281372, 0.0056388769298791885, -1.951241135597229, -0.033051785081624985, -1.1211414337158203, 0.447602242231369, 1.1177064180374146, 0.42236390709877014, -0.8761276006698608, 0.4854714870452881, -0.7657439112663269, -1.3621783256530762, 1.431968092918396, -0.7820685505867004, -0.7702775597572327, -0.6474501490592957, 0.9766714572906494, -0.3801622688770294, -0.31946367025375366, 2.0153913497924805, -0.7600283622741699, 0.27076733112335205, -0.31359827518463135, -2.4591996669769287, -0.509510338306427, -1.568317174911499, 0.45097440481185913, -0.11274503916501999, 2.5656518936157227, -1.724488615989685, 1.054512619972229, -1.824088454246521, -2.318513870239258, 1.0788148641586304, -0.8352591395378113, -0.7059997320175171, 0.7160831689834595, -0.6335238814353943, -0.13113580644130707, 0.8434416055679321, -0.8583420515060425, -0.596053421497345, 0.4090922772884369, -0.7325747013092041, 0.006057934369891882, -0.9390122890472412, 0.6466760039329529, -1.44728684425354, -0.3627534508705139, -0.52860426902771, -1.0345546007156372, 0.3009207844734192, -1.0271259546279907, -1.481927752494812, -0.32044172286987305, 0.8696189522743225, 0.47351017594337463, -0.6464351415634155, 0.6833010315895081, 2.023376941680908, 1.0947836637496948, 1.2812069654464722], [-0.7755885720252991, -0.6058678030967712, 0.9929158687591553, 1.6912881135940552, 1.7256407737731934, -0.04276810213923454, -1.777842402458191, -1.1575424671173096, -0.06369494646787643, 0.7321550846099854, 0.5268730521202087, -0.26661449670791626, -0.8415600061416626, 1.81902015209198, -0.6442456245422363, -0.6138550043106079, 0.36637216806411743, 0.2444494068622589, 0.5834032297134399, 0.1467842310667038, -0.11003720760345459, -1.1009429693222046, -0.2784987986087799, 0.031217580661177635, -0.27987557649612427, 1.653401255607605, 1.225398063659668, 1.4722291231155396, -0.33935925364494324, -0.6629577279090881, -0.9198135137557983, -0.8357883095741272, -0.9620816707611084, 0.6069846153259277, 0.7233615517616272, 0.9594693183898926, -1.1214457750320435, 0.12072321027517319, -0.5602055788040161, -1.2729904651641846, -2.056891918182373, -1.0569720268249512, -2.702043294906616, -0.2871937155723572, 0.04629138484597206, 1.3281891345977783, -0.531749427318573, 0.0033306023105978966, -0.10023034363985062, -0.48337891697883606, -1.846195101737976, -1.3516082763671875, -0.06804816424846649, 1.4578615427017212, 0.7246400713920593, -1.0101121664047241, -1.8759162425994873, -0.47689539194107056, 0.7772412896156311, 0.6537373661994934, 0.02092408761382103, 0.4037210941314697, -0.5318374633789062, -0.021107058972120285, 0.39966312050819397, -0.011455101892352104, 0.5621955990791321, -1.1321666240692139, 0.05826719105243683, 0.7517747282981873, 0.585223913192749, 0.5741796493530273, 1.0488296747207642, 0.03642966225743294, -0.4015233814716339, -0.11891394853591919, 0.32086318731307983, -1.0431783199310303, 0.7754513025283813, 0.36891382932662964, 1.6714104413986206, -2.3499577045440674, 0.7869411706924438, -1.145447015762329, 0.5364055633544922, -0.24785415828227997, -0.3493548631668091, 0.29971203207969666, 0.5556806921958923, 0.32275110483169556, 0.8605256676673889, 1.6286367177963257, 1.4144197702407837, -0.8883787989616394, -0.35825255513191223, -0.5494838356971741, -1.7039003372192383, -0.4921756088733673, -0.19105473160743713, 1.3257403373718262, -0.050529804080724716, -0.7239290475845337, 1.5610090494155884, 1.2652499675750732, -0.9270297884941101, -0.6825205683708191, -0.5363777279853821, -0.6066078543663025, 0.2325376719236374, -0.608187735080719, -0.41633787751197815, -1.159784197807312, -0.9172847867012024, -1.6426180601119995, 0.28869059681892395, -0.7488394379615784, -1.4657392501831055, -0.23268312215805054, 1.0674176216125488, -0.39412903785705566, 0.687496542930603, -1.9030612707138062, -0.005080171395093203, 0.7127326726913452, 0.9705179929733276, -1.7109169960021973, 0.2827013432979584, -0.16049227118492126], [-0.8512595891952515, -1.4832525253295898, -1.3799113035202026, 0.11701638996601105, 0.04898231849074364, -0.49194419384002686, 0.7198314070701599, 0.5114790201187134, 0.15525588393211365, -1.1514376401901245, -0.26235929131507874, -1.0409657955169678, 1.2256783246994019, -0.8028357625007629, 0.43460485339164734, -0.6308062076568604, 1.1899749040603638, 0.5494700074195862, 1.036320447921753, 1.3763271570205688, -1.7916172742843628, -1.3304523229599, -0.21797290444374084, 0.5526571273803711, 0.6551231741905212, 0.9375169277191162, -0.2656220495700836, 0.5995181202888489, -0.5437713265419006, 0.7210531830787659, -3.0542211532592773, 0.1384897232055664, 0.39545196294784546, -1.815185785293579, -1.402376651763916, -0.6301212310791016, 2.3656179904937744, -0.038340214639902115, -0.7071878910064697, 1.4013696908950806, 1.1894282102584839, -0.904835045337677, 1.4714422225952148, -0.33997026085853577, 0.673747181892395, 0.27626678347587585, 0.18523642420768738, 0.8704310059547424, 0.5354544520378113, 1.0359715223312378, -0.261593222618103, -0.7114453315734863, 1.585308313369751, -1.4542995691299438, -1.7373104095458984, 0.8950695991516113, -0.8925296068191528, -0.578245997428894, -1.1982632875442505, -0.32898661494255066, -0.5574461221694946, 0.21645331382751465, -1.8278896808624268, 0.5877248048782349, 0.8157082200050354, 0.3360314667224884, -0.43148401379585266, 1.312183141708374, -1.5839998722076416, -0.6244097948074341, -0.45914143323898315, 0.8418789505958557, -0.11158708482980728, 0.03287234902381897, -0.22291861474514008, -2.0569653511047363, -1.7079622745513916, 0.9295191168785095, 0.3434756100177765, 2.667203903198242, -0.830542802810669, 1.6796835660934448, -0.07501288503408432, -0.048315033316612244, -0.5381024479866028, 1.1287288665771484, -0.11488118767738342, -0.06251338124275208, 0.02571992017328739, -0.24181514978408813, 2.276219129562378, 2.6615798473358154, 0.3350813686847687, -1.4912446737289429, 1.4253872632980347, -1.1791914701461792, 0.7517557740211487, -0.31913653016090393, -0.8613904714584351, -0.5268792510032654, 0.5878986120223999, -1.3020070791244507, 0.1378503143787384, -2.2910499572753906, 0.761688232421875, -0.6376219391822815, 0.08950735628604889, 1.0959721803665161, 0.20404815673828125, -1.9504542350769043, 0.19103559851646423, -1.930547833442688, -0.5587866306304932, -0.1897784173488617, -0.9068087339401245, 0.2418977916240692, 0.7842845916748047, 1.7642241716384888, -1.6993366479873657, 0.35912811756134033, -1.9895060062408447, 1.003022313117981, -0.03534814342856407, 1.0486557483673096, 1.06252121925354, 0.4358513653278351, 0.06904204934835434, 1.3564718961715698], [1.1888186931610107, 1.510200023651123, -0.016233984380960464, -0.9670263528823853, -0.04846436157822609, -0.668207049369812, 0.08595050126314163, 1.6447077989578247, -0.5203574895858765, 0.2319440245628357, 0.3281884491443634, 1.4765725135803223, 0.8667529821395874, -1.0014586448669434, 0.5698375105857849, 0.4441101551055908, 0.5580248236656189, -0.3982506990432739, -1.33883535861969, 0.5233666896820068, -1.6995820999145508, 0.6632035374641418, -1.466622233390808, 0.09827712178230286, 0.07282103598117828, -1.5135823488235474, 0.655007541179657, 0.3248145282268524, 0.3308233916759491, -0.3449094295501709, 0.9734205603599548, -0.9975053668022156, -0.24277618527412415, 0.18647684156894684, 0.10189317911863327, 1.4304242134094238, -0.5377987623214722, -0.8504747152328491, -0.7927254438400269, -0.6111302971839905, 1.631112813949585, 0.9059238433837891, -1.0077520608901978, 0.9450963735580444, 1.584073781967163, 2.607649326324463, 1.1085364818572998, 1.126741647720337, 0.894403874874115, -0.6765761971473694, 0.18079181015491486, -0.9498281478881836, -1.251779317855835, 1.407076358795166, 0.4737246036529541, 0.9156631827354431, 1.4203084707260132, 0.784530520439148, -1.1947381496429443, 0.593430757522583, -0.4190453886985779, 0.12155667692422867, 1.6352804899215698, -2.332374095916748, 0.30329033732414246, -0.5978977084159851, -0.5183929800987244, -0.34750810265541077, 1.0254985094070435, 1.0514065027236938, 0.5864580273628235, -0.7499532103538513, 1.5051125288009644, -0.6498042941093445, -1.2741936445236206, 1.7069908380508423, -0.6724289059638977, 0.5451784133911133, -0.4968082010746002, 0.19492207467556, -1.6316916942596436, -0.9197803735733032, -0.487775593996048, 2.146949052810669, -1.339425802230835, 0.41300061345100403, 0.158030167222023, -0.05061165243387222, 0.9513518810272217, -1.0417696237564087, -1.1495741605758667, -0.330733984708786, 0.7455370426177979, -0.46988946199417114, 0.420663982629776, 0.7843276858329773, 0.4327229857444763, 0.1767827272415161, 0.029462819918990135, 1.1863504648208618, 0.2028505802154541, -0.16140343248844147, -0.8975338935852051, 1.391116976737976, -2.252265453338623, -0.9996607899665833, 0.027704311534762383, -0.979383111000061, 1.835443377494812, 0.44838911294937134, -0.8084964156150818, 1.5780987739562988, -3.0925281047821045, 0.3704058527946472, 2.0112979412078857, -0.05384261906147003, -0.9649662971496582, -0.2385166436433792, -0.30911991000175476, -0.34328413009643555, 0.3748653829097748, -0.060912735760211945, -0.5232218503952026, 0.1816224604845047, 1.3423882722854614, 1.835433006286621, -0.29160696268081665, -0.47109976410865784], [0.013153602369129658, 0.5922786593437195, -0.055639881640672684, 0.7977052927017212, -0.09989926964044571, -1.6063874959945679, 0.8943337798118591, 1.0857971906661987, 1.2878657579421997, -1.2353038787841797, 0.9090218544006348, 0.515180766582489, 0.7583444118499756, 0.36979055404663086, 1.2445590496063232, -0.07090506702661514, 0.3820823132991791, -0.5602009296417236, -1.8526049852371216, 0.46098655462265015, -0.4501524567604065, -1.7181761264801025, 1.7612476348876953, 1.3865253925323486, 0.48026013374328613, 1.0204780101776123, 0.8016967177391052, -0.09016003459692001, 1.4842652082443237, 0.06088104471564293, 0.40126287937164307, -0.08345276862382889, -1.0090221166610718, -0.19170236587524414, -0.5678954720497131, -0.801201343536377, 0.4629683792591095, -0.6011867523193359, -0.11526744812726974, 0.8952729105949402, -0.8006091117858887, 0.7119022011756897, -0.4360862374305725, -0.5223198533058167, -0.4479241669178009, -1.64491868019104, 0.352149099111557, 1.3582977056503296, -1.0340925455093384, -0.07258687168359756, -0.0924949124455452, -0.30133163928985596, 0.548008143901825, 1.0922516584396362, -1.4349404573440552, 0.15908250212669373, 1.26344895362854, -0.40247076749801636, 0.6324716806411743, 0.640804648399353, 0.23038600385189056, -0.632731556892395, -2.5668532848358154, 0.7268064618110657, 0.7247831225395203, -0.23894834518432617, 0.5330590009689331, 0.5717489719390869, 0.11267336457967758, -0.4302191734313965, -0.7105235457420349, -0.6632720828056335, 0.9855436682701111, 1.1015290021896362, 0.7479242086410522, -0.8025063872337341, 2.1330130100250244, 0.32557347416877747, 1.2714557647705078, 0.2693442106246948, 1.4721298217773438, 1.1917434930801392, -0.8163745403289795, 0.5557098388671875, -0.8269242644309998, -0.10514838248491287, 0.8210065364837646, 1.0649380683898926, 1.1851568222045898, 0.6763406991958618, -0.8717694282531738, -0.5596293807029724, -1.1703518629074097, 0.5016746520996094, 0.7088313698768616, -0.37622299790382385, -0.9327484965324402, -0.688585102558136, -1.2390025854110718, 0.39922142028808594, 0.4675590693950653, 0.07750292867422104, 0.09167835861444473, 0.8102647662162781, 2.120046377182007, 1.4446951150894165, 0.7452433109283447, 1.9710673093795776, 2.175226926803589, 0.16050729155540466, 1.1314693689346313, 1.03361976146698, 1.4191433191299438, -0.00042082747677341104, -1.2510461807250977, -0.27942946553230286, 0.3852246403694153, 0.8667516708374023, 0.583480715751648, 0.6412693858146667, 0.7772775292396545, 1.3001439571380615, 0.0175215732306242, 0.31569692492485046, -1.6232914924621582, -1.4040882587432861, -1.1006008386611938, 1.1710407733917236], [0.4626877009868622, -0.06175512447953224, 0.6062898635864258, -0.9416003823280334, 1.1537985801696777, 1.1497395038604736, -0.2175460010766983, -0.973650336265564, 0.6289952993392944, 0.9855511784553528, 0.6151349544525146, -0.6968193054199219, -0.3912861943244934, -0.6297159194946289, -1.149755597114563, -0.18522000312805176, -0.026187824085354805, 2.2630536556243896, 0.14473854005336761, -0.3210659325122833, 1.7241536378860474, 0.5714651346206665, -0.5737324357032776, -0.04711338132619858, -0.05224189534783363, -0.18311944603919983, -0.10231756418943405, -0.33718371391296387, -0.9338374733924866, -0.2157183140516281, 2.1185476779937744, -0.6768983602523804, 0.47234007716178894, 0.7485247254371643, 0.04953569173812866, -0.9361969232559204, -1.1926870346069336, 0.8077505826950073, 0.17705203592777252, 0.48857611417770386, -1.115918517112732, 0.7779034972190857, -0.36479389667510986, 0.8225830793380737, 0.18097801506519318, 0.00016036038869060576, -2.1697986125946045, -0.7556877732276917, 0.5907092094421387, 1.605920672416687, -2.1749215126037598, 0.31884121894836426, -0.18192048370838165, -0.41461578011512756, 0.4265362620353699, 0.5390154123306274, 0.35046494007110596, -0.37473371624946594, -1.1280452013015747, -0.4122588038444519, -0.4848516285419464, -0.7505150437355042, 1.2069720029830933, -1.1738847494125366, -0.14754527807235718, -0.6228083372116089, -1.5331436395645142, -1.0867080688476562, 0.498722106218338, -0.18425726890563965, 0.004901425447314978, -1.52838933467865, 0.785789430141449, -0.6589211821556091, -1.5366359949111938, 0.39800575375556946, -0.11278124898672104, 0.8551622033119202, -0.2970970869064331, 0.42564302682876587, 0.906102180480957, 0.02308727614581585, 0.10102414339780807, -1.5368903875350952, 0.6119912266731262, 1.749670147895813, 0.9814237356185913, -0.12348310649394989, -0.19469384849071503, -1.130314588546753, 0.45889338850975037, -0.33373817801475525, -0.4863424003124237, -0.2569587230682373, 0.6567468643188477, -0.1443140059709549, 0.647693932056427, 0.943807065486908, 0.5616965889930725, 0.06129618361592293, 0.7141837477684021, -0.2760654389858246, -1.214898705482483, 0.16473056375980377, 1.2212600708007812, -0.35547319054603577, 1.9526927471160889, -0.10621006041765213, -0.7006590366363525, -0.25091949105262756, 0.7234156727790833, -1.3707162141799927, -0.5831793546676636, -0.6686183214187622, 0.1708011031150818, -0.695424497127533, -0.5034520626068115, 1.5011358261108398, -0.032565467059612274, -0.5592718124389648, 1.3977289199829102, -0.02419099025428295, -0.024346528574824333, 1.1704494953155518, -1.0814428329467773, -1.170855164527893, -0.25985634326934814, -0.026871243491768837], [-0.22502394020557404, 0.8453071713447571, 0.6893552541732788, 0.48025357723236084, 0.3113081157207489, -0.6476648449897766, -0.11748262494802475, 1.4222474098205566, 0.4079391658306122, 0.8305455446243286, -1.0673301219940186, -1.049776315689087, -1.4342639446258545, -0.44410941004753113, -0.135612353682518, 1.0098240375518799, 0.11070409417152405, 1.1991029977798462, 0.11807677149772644, 0.10666394233703613, 0.3469911217689514, 2.3528120517730713, 0.3498556613922119, -1.086785912513733, 0.3800041377544403, -0.08521867543458939, 1.066457986831665, -0.5922365784645081, -0.35221970081329346, -0.8555333614349365, 0.3747764825820923, 0.8665913939476013, -1.0781913995742798, -0.09707563370466232, -0.6176461577415466, 0.9463900327682495, -0.6816084980964661, -1.1742295026779175, -0.3277575373649597, -0.6398023366928101, -0.3607572019100189, -0.06709703803062439, 0.20823398232460022, 1.092605471611023, -0.10686397552490234, -0.974311888217926, -0.5268738865852356, -3.8489863872528076, -1.0092166662216187, 2.713569402694702, 0.17735454440116882, 2.6142594814300537, -0.479949027299881, 1.0630680322647095, -0.11796317249536514, 0.07710090279579163, 0.4835222363471985, -1.7366987466812134, 1.4603527784347534, -0.14385318756103516, -1.035217046737671, -1.0208247900009155, 0.9860464334487915, -0.31630203127861023, -1.3121411800384521, 0.423970103263855, 1.0326125621795654, -1.6755292415618896, 1.2142045497894287, -2.0621442794799805, 0.8226104974746704, 0.74651700258255, -1.2360758781433105, -1.3328421115875244, 0.4498940110206604, -0.35853615403175354, -0.06202654168009758, -0.5485544204711914, 1.1205850839614868, 1.6049495935440063, -0.11542943120002747, 0.6175040602684021, 0.8213173151016235, 0.5290074944496155, -0.2317289561033249, 1.0084813833236694, 0.7422699928283691, 0.8070681691169739, -1.1994433403015137, -1.2013112306594849, -2.21671724319458, -0.2218593806028366, -0.6233180165290833, -0.7062830924987793, -1.6536521911621094, 0.3235296607017517, 0.1978411078453064, 1.180867314338684, 0.6179491281509399, 2.0886974334716797, -2.3970322608947754, -1.9182645082473755, -1.428162932395935, -0.4851841330528259, -2.5302770137786865, 0.22865775227546692, -0.1289810985326767, 2.472059488296509, 1.173064947128296, -0.610492467880249, 1.1762276887893677, 0.5129022598266602, -0.7605288028717041, 0.10777793824672699, -0.5399295687675476, -0.03292069211602211, 0.8691064119338989, -1.3711662292480469, 1.5670497417449951, 0.35689854621887207, 0.18200631439685822, 1.01231849193573, 0.2338404804468155, -0.7451627850532532, 1.3695930242538452, -0.1613597273826599, 0.2824607491493225, -0.5674633979797363], [-1.1315298080444336, -1.2215577363967896, -1.2732186317443848, -1.1051093339920044, -0.2678360939025879, -0.2808532118797302, 0.49880269169807434, 0.7343703508377075, -1.3580635786056519, -0.8462104797363281, 0.2599889039993286, -1.09890878200531, -0.034530967473983765, 0.0837421864271164, 0.3102380335330963, 2.389359951019287, 0.8658844828605652, 0.007049919571727514, 1.3018368482589722, 0.893126904964447, 0.9744853377342224, -1.7536991834640503, -1.5112555027008057, -1.9449294805526733, -1.9393924474716187, -0.18658308684825897, -0.8529738187789917, 1.399490475654602, -0.6224386096000671, 1.593950867652893, 0.4707065224647522, 0.4556131362915039, 1.085095763206482, -0.5002882480621338, 0.6215729713439941, 0.1581135243177414, -0.37065207958221436, -1.4604476690292358, 0.6516285538673401, 1.0018974542617798, 0.37279805541038513, 0.9198682904243469, 1.1255931854248047, -0.48908859491348267, 0.4247822165489197, -0.4393062889575958, -0.19056281447410583, -1.2978265285491943, -1.220699429512024, 0.3500536382198334, 1.8008896112442017, -0.30235135555267334, 0.5962023735046387, -0.36405906081199646, -0.6188050508499146, 0.5708637833595276, -0.49769023060798645, -0.24226613342761993, -0.2995574176311493, 0.13610225915908813, 0.7909199595451355, 0.21798193454742432, -0.4102500081062317, 0.9632558226585388, 0.6284383535385132, 1.2537569999694824, -0.4216001331806183, 1.0850303173065186, 0.038922056555747986, 1.3456203937530518, -0.3610166013240814, -1.4092328548431396, -0.18143697082996368, -0.10257736593484879, -1.042030692100525, 1.1684101819992065, 0.9198386669158936, 1.098202109336853, -0.23814234137535095, -1.3343734741210938, 1.4291112422943115, -0.39287006855010986, 0.46494221687316895, 0.01224072277545929, 0.6118326783180237, -0.3539482355117798, 2.0506489276885986, -0.9347587823867798, -1.9446412324905396, 0.8829346895217896, 0.9508246183395386, 0.4488998055458069, 1.6650121212005615, -1.7951111793518066, 0.912803053855896, -0.4752140939235687, 1.0937950611114502, -1.0583984851837158, -0.07374420017004013, -0.27163729071617126, -0.09826397895812988, -0.7623158693313599, 2.133103370666504, 0.11623212695121765, 0.19806653261184692, 0.07415284216403961, 0.8290308117866516, 2.365776300430298, 0.49146032333374023, 1.3189327716827393, 0.22141340374946594, 2.3849472999572754, -1.078192949295044, -0.3628138303756714, -0.11222352087497711, 0.36470693349838257, 0.5487961769104004, 1.0992454290390015, 0.008114888332784176, 2.401400566101074, 0.5344338417053223, -1.1723244190216064, -2.225773334503174, -0.7844481468200684, -0.24869178235530853, -0.42182403802871704, -1.0663461685180664, -2.154005289077759], [-0.029457496479153633, -1.414872169494629, -0.15867654979228973, -0.07014855742454529, 0.32859736680984497, 0.013409286737442017, -0.38907283544540405, -0.4761391580104828, 0.2402035892009735, 0.3247070908546448, -0.3780222535133362, -0.6020851731300354, 0.5880078673362732, -1.6638809442520142, -0.9164437651634216, -0.25794368982315063, 0.09914592653512955, 0.14668092131614685, -0.9828732013702393, -0.12000666558742523, -0.6385849118232727, -0.38374772667884827, -0.4475115239620209, 1.9827990531921387, 0.35085415840148926, 1.312662959098816, 0.8457545638084412, -0.004440664779394865, -1.0156581401824951, -1.6272531747817993, 1.0606900453567505, 0.4680362343788147, -0.05868348479270935, 0.1499023288488388, -1.7501404285430908, 0.33455848693847656, 0.2684917449951172, -0.9166257977485657, 1.6360641717910767, -0.30489927530288696, -0.6818612217903137, 0.2238350659608841, -0.7316038608551025, 0.4679708778858185, -2.26069974899292, 0.5032204389572144, -0.6475532054901123, -2.448660135269165, 0.9252369403839111, -1.9971321821212769, 0.6392831802368164, 0.10867702960968018, -0.711327850818634, -1.482113242149353, 0.25430065393447876, -0.1658930778503418, 0.8388254046440125, 1.804408311843872, 0.6709738373756409, 0.39567869901657104, 1.0292906761169434, 0.5077968239784241, -0.08203314244747162, 1.4876714944839478, -1.4059067964553833, 0.6386559009552002, -0.919012725353241, 0.2898748517036438, 0.24877715110778809, 0.023213759064674377, 1.4640758037567139, -0.4413793087005615, 1.0612612962722778, -1.900916576385498, -0.5524234175682068, 0.3099070191383362, 0.54441237449646, 1.0827865600585938, 0.9109528660774231, 2.018470525741577, 0.719594419002533, -0.3984307646751404, 0.1076289564371109, -0.16362784802913666, -0.18243102729320526, 1.5193579196929932, 1.6782747507095337, -0.10071813315153122, 2.0571048259735107, 0.9756865501403809, 0.5309285521507263, -0.050085075199604034, -0.14393427968025208, 1.089539647102356, -1.144593596458435, -0.4718865752220154, -0.8191481232643127, -0.3067112863063812, 2.324105739593506, -0.8636789917945862, -0.32417458295822144, 0.4531804621219635, -0.41455456614494324, 0.3110645115375519, 0.5848391652107239, 0.7539802193641663, 1.610719084739685, 1.480376124382019, 1.0610322952270508, -2.016252279281616, 0.35597923398017883, -1.1336796283721924, -0.7142131924629211, -0.993316113948822, -0.8485131859779358, 0.5618181824684143, -0.020183993503451347, 0.516292929649353, -1.26505708694458, -1.2363455295562744, -0.16458453238010406, -0.785377025604248, 0.08689778298139572, -1.3947534561157227, -0.5145244002342224, 0.08667608350515366, 1.186539649963379, -0.5139943957328796], [1.8635222911834717, 0.5498991012573242, -1.4855214357376099, -1.0084772109985352, 1.2780920267105103, 1.0118417739868164, -0.9424446821212769, 0.18649455904960632, 2.4975967407226562, 0.09641128033399582, -0.45249953866004944, -0.9415874481201172, -1.070381999015808, 1.0707411766052246, 0.8269683718681335, 0.17545713484287262, -0.18402735888957977, 0.2211751490831375, -2.268141508102417, -2.9527580738067627, -2.1322474479675293, -0.48835501074790955, 0.6247121095657349, 1.4316946268081665, -0.04024891555309296, 0.3653498888015747, 0.8153764605522156, -1.2064203023910522, -0.5489132404327393, -0.03344129025936127, 0.018750177696347237, -1.5196083784103394, 0.6844313144683838, -0.03621596097946167, 1.0516637563705444, -0.2787284851074219, -1.1148124933242798, -0.8986481428146362, 0.4544655382633209, 1.4710376262664795, -1.5076050758361816, 1.8623892068862915, 1.1222662925720215, -0.7962110638618469, -0.9386162161827087, -0.1570175439119339, 0.2653570771217346, -1.7095173597335815, 1.531137228012085, -0.8614873290061951, 1.2362371683120728, 1.0577958822250366, 0.1794072985649109, -0.5911920070648193, -1.2373383045196533, -0.5569965243339539, -0.19079287350177765, 1.0948610305786133, 0.41387930512428284, 0.056538451462984085, 0.36057591438293457, -0.09824575483798981, -0.02129487507045269, -1.7312476634979248, -1.2829045057296753, 0.6650561690330505, 0.5834290385246277, -0.6754665970802307, 0.8168312907218933, -1.6762202978134155, 0.6809033751487732, 0.73759925365448, 1.570361852645874, -0.22178351879119873, 0.09649024158716202, 0.7143650054931641, -0.11658559739589691, -0.9736907482147217, -0.10650845617055893, -1.7904489040374756, 0.2128356695175171, -0.8901856541633606, 0.29144948720932007, 1.1455415487289429, -0.19325900077819824, 1.5262999534606934, -1.3983100652694702, 1.1510180234909058, -0.18388248980045319, -0.15162092447280884, -0.12197603285312653, 0.02037326619029045, -0.6284776329994202, 1.2822775840759277, 0.13827969133853912, -0.021772101521492004, -0.5881026387214661, 1.759079933166504, -1.4407923221588135, 0.3865397870540619, -1.9052879810333252, 1.7633671760559082, 1.673324465751648, 0.7358869910240173, -0.06877369433641434, 0.03287040442228317, -1.0234100818634033, -0.03655502200126648, -1.2047955989837646, 1.285186767578125, 0.421752393245697, 0.8548610210418701, 0.5797402262687683, 0.7803260684013367, -0.7971253991127014, 0.2958759367465973, 0.6703137159347534, -0.006160592660307884, 0.6359078884124756, -0.3791073262691498, 0.5501931309700012, 0.5319703221321106, -0.4019240736961365, 0.37171056866645813, 2.0820627212524414, -0.5408459305763245, 0.12178362905979156, 0.4330950677394867], [0.9623730778694153, 0.8488894701004028, -0.20847804844379425, -0.4364154636859894, 0.7623011469841003, 0.27520933747291565, 0.0014358505140990019, -0.629276692867279, -0.39280450344085693, -0.4433525800704956, 1.5558900833129883, 1.1539417505264282, -1.9820597171783447, 0.8491525053977966, -1.3822747468948364, -1.429681658744812, 1.3836853504180908, 1.486114501953125, -1.257161021232605, -2.4233529567718506, 1.5956993103027344, -0.24681028723716736, -1.0759676694869995, -1.8184360265731812, -0.020867090672254562, 1.0950605869293213, -0.7556942105293274, 0.33502617478370667, 0.017593543976545334, -1.4805312156677246, -0.9817401766777039, -0.6639732718467712, 0.7054996490478516, 0.8430245518684387, 0.47947824001312256, 0.36924245953559875, -1.0099689960479736, -0.33163267374038696, 0.6848781108856201, 1.0751895904541016, -0.22537101805210114, 2.4396681785583496, 3.4119038581848145, 0.03114846535027027, -1.3174208402633667, -0.43785253167152405, 0.5067116022109985, -0.24142912030220032, -1.0949747562408447, 0.9424844980239868, 1.419789433479309, 1.460631012916565, 0.44872018694877625, 0.8372027277946472, 0.7950998544692993, -3.0929410457611084, 0.8164490461349487, -1.2125732898712158, 1.251942753791809, -0.1970561146736145, -0.7815788984298706, -0.5592177510261536, 1.1634275913238525, -0.37524813413619995, 1.830174207687378, 0.14158597588539124, -0.8736253380775452, -1.2968225479125977, 1.420419454574585, -1.0959725379943848, -0.6824131608009338, -0.12949249148368835, -0.45498788356781006, -0.23915350437164307, -1.562518835067749, -0.7261084914207458, -0.8012413382530212, -0.7504720687866211, -0.013656919822096825, 0.3688300549983978, -1.2505168914794922, 1.20072603225708, 0.5578180551528931, 0.24924428761005402, 1.0332653522491455, -1.143545389175415, 0.9613397717475891, -0.4674326181411743, 0.5614878535270691, -0.40729278326034546, 0.07457631081342697, 1.5621626377105713, -0.412948340177536, -0.5773804783821106, -1.0840116739273071, -0.45316046476364136, -0.9627107381820679, 0.37551945447921753, 0.5606573820114136, 0.4714665710926056, -0.42177385091781616, -0.5810952186584473, -0.9260896444320679, -0.01927102915942669, 2.125431537628174, 0.6469972729682922, 0.1742473989725113, -2.3779683113098145, 0.9133010506629944, -0.5671700835227966, -1.3121283054351807, -1.6762118339538574, -0.13929153978824615, 0.482930988073349, 1.2535960674285889, 0.052603159099817276, -0.7061067223548889, -1.0677281618118286, -0.0885361060500145, 1.2280387878417969, 0.7169011831283569, 0.18142856657505035, -0.35330966114997864, -0.44657936692237854, 0.105679951608181, 0.1791684329509735, -0.08078961074352264, -0.17606917023658752], [-0.8188232183456421, 1.9787116050720215, -0.09422978013753891, -0.4606245458126068, -0.8636672496795654, -0.9158102869987488, -1.3270647525787354, 1.090197205543518, 0.23804853856563568, 1.1311707496643066, -2.015401840209961, -1.5077872276306152, 0.7764330506324768, 0.5303319096565247, 0.5355815887451172, -0.8626717925071716, 0.5909442901611328, 0.5326366424560547, -1.4994890689849854, 0.8291932940483093, -1.0275501012802124, 1.4563746452331543, -0.965617835521698, -1.2475937604904175, -1.0363013744354248, -0.3493744730949402, 1.4397894144058228, 1.6387625932693481, -0.3269926607608795, 1.9109039306640625, -0.1750553399324417, -1.3678663969039917, -1.5130226612091064, -1.717705488204956, -1.0546300411224365, 1.6168320178985596, 0.8388783931732178, -0.19162890315055847, -0.1466141641139984, -1.0461128950119019, -0.4629126489162445, -0.7767021656036377, -0.8572418689727783, 1.849583387374878, -0.03563161939382553, -1.5618523359298706, -0.5914554595947266, 1.5506997108459473, -0.4604915678501129, -1.875043511390686, -0.3387037515640259, 0.1503899097442627, -0.29489487409591675, 0.538129448890686, 0.16828647255897522, 0.35127830505371094, 0.3605463206768036, 1.5082565546035767, 0.20722568035125732, 1.1120802164077759, 1.6155036687850952, -0.9913880228996277, 0.14624224603176117, 0.0065181502141058445, -1.0078152418136597, -0.7093846201896667, 1.5757684707641602, 0.11677480489015579, 0.05839161202311516, 1.129698395729065, 1.3280552625656128, -1.4388539791107178, -0.4714099168777466, 0.9456061124801636, -2.079953193664551, 0.4075712561607361, 0.8455086350440979, -1.0198729038238525, -1.0644400119781494, -0.5279465317726135, 1.6933811902999878, 0.010545467957854271, 0.789695143699646, 1.056532382965088, -1.4041965007781982, -0.018704552203416824, -0.7606413960456848, 0.7556580901145935, -1.0586681365966797, -0.5632861256599426, 2.821164131164551, 0.10250364243984222, -0.9539315700531006, 1.396005392074585, -0.9410228133201599, 0.7067270874977112, 0.27987465262413025, 1.143181562423706, -0.4301862120628357, 0.30776268243789673, -0.5616717338562012, 1.3312991857528687, 0.4619859755039215, 0.3774961233139038, 1.6502865552902222, -0.0567195788025856, 0.8530719876289368, -1.5482500791549683, -0.6203118562698364, 0.42599084973335266, -0.07222530245780945, 0.2571491003036499, -0.22555963695049286, -0.17848651111125946, -0.9841052293777466, 0.6197176575660706, -1.8321388959884644, -0.3724214732646942, 0.37527987360954285, 0.22258158028125763, -1.0697249174118042, 0.8180323839187622, -1.196822166442871, 0.49807941913604736, 0.8843483328819275, 0.15413005650043488, 0.6530773639678955, -0.44128844141960144], [-0.31801897287368774, -1.1036125421524048, -0.008039710111916065, 0.4871252179145813, 0.16911263763904572, -0.6900609135627747, -1.5398586988449097, -0.6389157772064209, 0.07995574176311493, 1.7008739709854126, 0.11359170079231262, 1.1600443124771118, -0.7336342334747314, -1.701720952987671, -0.22441674768924713, 0.48428118228912354, -1.1113910675048828, -1.3000564575195312, -0.9368041753768921, -0.9911453723907471, -0.072026327252388, -0.5912903547286987, -0.230865478515625, 0.006141458172351122, -1.9232298135757446, -2.011746644973755, 1.306422472000122, -0.3461686372756958, 1.5930334329605103, 1.4881341457366943, -0.6408027410507202, 0.8951502442359924, -0.06070427596569061, -0.3250686824321747, 1.0926052331924438, 2.084240198135376, -0.3136957883834839, -2.1460025310516357, -1.3568307161331177, -1.8621282577514648, 0.7359263896942139, -1.5396336317062378, -0.551045835018158, -1.6235370635986328, -2.1307501792907715, -0.6555992364883423, 0.8474393486976624, 0.03790992498397827, -2.101142406463623, -1.0010722875595093, 0.7048099637031555, 0.01959092542529106, 0.2197246104478836, -0.2630089521408081, -0.4736562967300415, -0.630561113357544, -0.8432883024215698, 0.29562482237815857, -1.7262948751449585, -0.8922199010848999, -0.8431394100189209, 1.5112061500549316, 0.3714594542980194, -0.7084863781929016, -0.6240357160568237, 0.6588244438171387, -0.40133196115493774, 0.8884839415550232, 0.15948042273521423, 1.0478399991989136, -0.09659262746572495, -0.9297373294830322, -0.13262030482292175, -1.3349111080169678, 1.652390480041504, -0.1595809906721115, 0.2933404743671417, 1.3892539739608765, -0.24489019811153412, 0.5230438709259033, 0.9038320183753967, 1.6996501684188843, 0.5281548500061035, -0.14230014383792877, 0.16291216015815735, 1.4831643104553223, -0.9317942261695862, -0.8603411316871643, -2.4217476844787598, -0.4417811334133148, -1.5756224393844604, 0.5050252676010132, 0.8994033932685852, -1.4108120203018188, -1.3993635177612305, -0.5784168839454651, 0.022403929382562637, 1.5081267356872559, 0.5470779538154602, 1.2004640102386475, -1.935034155845642, -0.5582593083381653, -0.08539003133773804, -1.0855094194412231, -1.2102218866348267, 0.1400539129972458, 0.7929991483688354, -1.639716625213623, -0.9723058342933655, -1.0127884149551392, -0.13226550817489624, -1.7690398693084717, 1.1267825365066528, -0.0437639020383358, -0.24800734221935272, -0.28809696435928345, 0.3466580808162689, 3.272193431854248, 0.09291601926088333, -0.7209922671318054, 0.9188255667686462, 1.6554951667785645, 2.0033113956451416, -0.644845724105835, -1.0967185497283936, 0.6193760633468628, -0.35302993655204773, 0.7737919688224792], [-0.9235920310020447, -0.9434120655059814, -1.392369270324707, 1.121032953262329, -0.17533767223358154, 1.160327434539795, -1.2604784965515137, 0.12315990030765533, -0.6787019371986389, -0.9216710925102234, 1.4177254438400269, -0.3438977003097534, 0.5636471509933472, -0.04842868819832802, 1.3256069421768188, 0.6962624192237854, -0.5798783898353577, 1.1160516738891602, -1.2779191732406616, 1.4076995849609375, -0.3668489158153534, 1.9047009944915771, 0.4347226023674011, 0.31679093837738037, -0.7450881004333496, 1.661648154258728, 0.24813687801361084, 1.063033103942871, 0.616283118724823, -0.6840466260910034, 0.6052418351173401, 1.364430546760559, 1.02117121219635, -0.3638099730014801, 0.6156219244003296, -1.9313840866088867, 1.4484177827835083, -0.42673155665397644, 0.30991414189338684, 0.3964586853981018, -0.12637637555599213, -1.2859896421432495, 0.44660061597824097, 1.1982039213180542, 0.903770923614502, -0.03786173462867737, 1.1301579475402832, 0.41285189986228943, 0.7046777606010437, -0.6244146227836609, 0.632455587387085, 0.15457943081855774, 2.6694247722625732, -1.234229326248169, -1.073921799659729, -0.7912549376487732, -0.9267538189888, -0.37097856402397156, 0.23852872848510742, -0.5851861834526062, -0.006190600339323282, -1.0034615993499756, 0.555461049079895, -0.9364922046661377, 1.0433716773986816, -1.1599016189575195, 0.39812955260276794, 0.07313097268342972, -1.97102952003479, 0.4731876254081726, -0.1701221466064453, -1.6154855489730835, 0.6171606779098511, -0.3628816604614258, -0.8912054300308228, -0.4364698529243469, -0.3474838137626648, -1.7388705015182495, 0.03966455161571503, 0.15233354270458221, -0.20705987513065338, -0.5423173308372498, 0.8259856104850769, -0.06605633348226547, 0.15932585299015045, -0.44501057267189026, -0.7993577718734741, 0.6343849301338196, 1.206291913986206, 1.06146240234375, -1.3963004350662231, 0.7882596254348755, 0.18028990924358368, -2.241177558898926, 0.09166588634252548, -0.9228873252868652, -1.6655614376068115, 0.37338075041770935, 2.179100275039673, -0.5909691452980042, -0.7502944469451904, 0.324327677488327, -0.05238179489970207, -0.4858740270137787, -1.466744065284729, 0.4269223213195801, -0.6924116015434265, 0.5843307971954346, -0.3540024757385254, -0.7405219674110413, -0.045486729592084885, -1.2834978103637695, 2.6355159282684326, -0.07793285697698593, -0.21278643608093262, 0.5621623396873474, 0.1264796406030655, -0.7645182609558105, -0.3633672893047333, 0.8312807679176331, 1.2137856483459473, -2.317164182662964, -0.5326932072639465, 1.6668912172317505, 0.6914647817611694, 1.45122230052948, 0.6735293865203857, 0.5784791707992554], [-0.3087235987186432, 1.2899099588394165, 1.308087944984436, 2.4236252307891846, -1.0030094385147095, -0.521787703037262, -0.32079142332077026, 1.216290831565857, 0.3688278794288635, 0.9563932418823242, 0.544840931892395, -1.1230716705322266, 0.2518806457519531, -1.2367115020751953, -0.44508856534957886, 0.6166369318962097, -0.046321846544742584, -0.5914353728294373, -0.5666512846946716, -0.1433982402086258, 0.014075271785259247, -1.5929516553878784, 0.5079955458641052, -1.182711124420166, -0.6151642799377441, -0.5255050659179688, 0.6378269791603088, 0.36145877838134766, 0.8846986293792725, -0.6495583057403564, 1.0804152488708496, 1.134125828742981, -1.1944164037704468, 0.2882539629936218, 0.05972208455204964, -2.727656126022339, 0.5210485458374023, 1.075669765472412, -0.3386518955230713, 0.20255008339881897, 0.34619319438934326, -0.9226126670837402, -0.4730491042137146, -0.36094048619270325, 0.06767819821834564, 0.20998995006084442, 0.7614790201187134, -0.6422020792961121, 1.3473390340805054, 0.41755348443984985, -0.06436853855848312, -0.49093538522720337, 0.7105376720428467, 0.9599736332893372, 0.6915260553359985, 1.9068697690963745, 0.1417267769575119, -0.030950386077165604, -0.27815884351730347, -0.8837284445762634, 0.29086601734161377, -0.5086202025413513, 0.1056889146566391, 0.9163206219673157, -0.13226142525672913, 0.5770536661148071, -1.5772055387496948, -0.10839953273534775, -0.7594778537750244, -1.439893364906311, 0.1987556666135788, 0.4748108983039856, 0.9745783805847168, -0.10561306029558182, 0.24695627391338348, 0.6499912142753601, -2.086442232131958, 1.2264851331710815, 2.1056737899780273, 0.7154433131217957, -1.2974530458450317, -0.02015487477183342, 1.4790043830871582, -0.5820042490959167, -0.3859499990940094, 0.6303905248641968, -1.7296775579452515, -0.273853063583374, 1.0955160856246948, -1.8609706163406372, -1.7444310188293457, -0.602727472782135, 1.6495269536972046, -0.17067451775074005, 1.528654932975769, 1.333723783493042, -0.06444180756807327, -1.2108960151672363, 0.23500359058380127, 1.5028330087661743, 1.4673856496810913, 0.03269689902663231, -0.0932120755314827, 0.6792684197425842, -0.6841886043548584, 1.233616828918457, -0.3281595706939697, 1.2405844926834106, -1.1725287437438965, -0.546207845211029, -0.4449257552623749, -1.8537406921386719, 0.7488391399383545, -0.5216570496559143, -0.43284910917282104, -0.26264506578445435, -1.6199015378952026, 0.04018792137503624, 1.9319415092468262, -0.8505977392196655, 1.7771015167236328, -0.29231300950050354, -0.428227961063385, 1.9518989324569702, 0.5775654315948486, -0.9586576819419861, 0.40405982732772827, 0.35480669140815735], [0.3243415951728821, 0.015437308698892593, -0.09570472687482834, -2.202720880508423, 0.5065011382102966, 4.167205810546875, 0.3222767412662506, 0.3097340166568756, 2.0728600025177, 0.1458677053451538, 0.7050414085388184, 0.9325432777404785, 0.4285818338394165, -1.2576614618301392, -0.38991615176200867, 1.2945398092269897, -0.7576549053192139, -1.6155023574829102, 0.4867538809776306, 0.7687456011772156, 1.5648561716079712, -1.0581614971160889, 0.11449185013771057, -1.6712491512298584, -1.3647114038467407, -0.123384490609169, 0.1966504156589508, 0.14156220853328705, -0.7830835580825806, 0.950206995010376, -0.8421997427940369, -0.31880080699920654, -0.1700449436903, -1.2859433889389038, -0.734442949295044, -0.11285139620304108, 0.6634682416915894, 0.6672040820121765, -0.21611660718917847, 0.25735917687416077, -0.09384065866470337, 0.4687565863132477, 2.814225673675537, 0.2253987342119217, -1.962662696838379, -2.4763550758361816, -0.9848080277442932, 0.46076375246047974, -1.2646093368530273, -0.8562649488449097, 0.15180528163909912, 1.2735260725021362, 0.2575734853744507, -0.4386405050754547, -0.22387614846229553, 0.7434605956077576, -0.20863784849643707, -0.72197026014328, 1.9498119354248047, 0.36992064118385315, -1.6504071950912476, 0.45400136709213257, -1.0539921522140503, -1.4754358530044556, 0.11133425682783127, -0.8898624181747437, 0.42923253774642944, 0.9539706707000732, -0.7833917140960693, -0.08100719749927521, -1.7316176891326904, -1.0578261613845825, 0.5442155599594116, 0.4952246844768524, 0.8750633001327515, -0.784694254398346, -0.6582067608833313, -0.3164607882499695, 2.613175392150879, 0.17624260485172272, -0.7669767737388611, -2.5959842205047607, 0.6415932774543762, 0.18578225374221802, 1.032446265220642, 1.1683748960494995, -1.7117847204208374, 0.9070557951927185, -0.4800270199775696, -0.2874332070350647, 0.5064648985862732, -0.22247987985610962, -1.3771430253982544, -0.5197968482971191, -0.525039553642273, 1.0136723518371582, -0.5220993757247925, 0.1264733076095581, -1.1013708114624023, -1.0318732261657715, -0.9334129691123962, 1.3450418710708618, -0.5192223191261292, -0.1742355227470398, -1.1003037691116333, 0.694823682308197, 0.42464590072631836, -1.0533246994018555, 0.9013418555259705, -1.7840677499771118, 0.6588186025619507, 0.07556045055389404, 0.6884803771972656, -2.1083824634552, -0.6426331996917725, -0.07480666786432266, 0.1647985428571701, -0.777332603931427, 0.9196300506591797, -1.271220088005066, 0.19921785593032837, -0.43695876002311707, 1.4977692365646362, -1.6666020154953003, 0.34179699420928955, 0.33089831471443176, 0.47524023056030273, -0.6637739539146423], [-0.4087359607219696, 0.5858264565467834, 1.8779109716415405, 1.4064258337020874, 0.8744725584983826, -0.8234686851501465, 0.9641016721725464, -1.772456407546997, 1.069523572921753, 0.33772924542427063, 0.2791405916213989, -0.1035822182893753, 1.3801378011703491, -0.3790919780731201, 0.06494453549385071, -0.34487542510032654, -1.1835650205612183, 1.2069586515426636, 1.4880539178848267, 0.5369648337364197, 1.4493578672409058, 0.037560220807790756, -0.5091078877449036, 2.1363210678100586, -0.47469040751457214, -1.0743041038513184, -1.6888624429702759, -0.3771705627441406, -1.1688348054885864, 0.21064263582229614, 0.8502121567726135, 0.29488006234169006, -1.7399625778198242, -0.3764920234680176, -0.7747926712036133, 0.17082200944423676, -0.35309576988220215, 1.1812283992767334, 1.2910736799240112, 0.7329305410385132, -1.057878851890564, -0.1768231838941574, -2.3427584171295166, 0.04857964068651199, -0.7992382645606995, -0.5570293068885803, -0.6898593902587891, -3.0253612995147705, -0.3637806177139282, -0.4489816725254059, 1.4348936080932617, 0.33219999074935913, -0.7882685661315918, 0.46741983294487, -0.31012779474258423, -1.0972509384155273, -0.557235836982727, 0.968801736831665, 0.05382164567708969, -1.0107626914978027, 0.3026731312274933, -0.23979628086090088, 0.4751669764518738, -1.0192539691925049, 1.0590109825134277, -0.11826649308204651, -1.1585652828216553, 0.39137667417526245, -0.111324243247509, -1.4501290321350098, 0.06549480557441711, 0.5050642490386963, 1.6219700574874878, 0.4931613802909851, 1.1999744176864624, -0.2895985543727875, 0.7743031978607178, 1.5873774290084839, -0.2309631109237671, -0.12484928220510483, -0.4270157814025879, 0.25034207105636597, 2.307560920715332, 1.5554507970809937, -0.06940901279449463, 1.7211040258407593, -1.4119458198547363, 1.6008775234222412, 0.11161316931247711, -0.5324811339378357, -0.8956053853034973, 0.12042263895273209, -0.7703935503959656, -0.7995144724845886, -0.6607893109321594, -0.2617887258529663, 0.8665224313735962, 0.798740029335022, 0.6709139347076416, -0.5474250912666321, -0.8626905679702759, 0.12143433839082718, -0.08427655696868896, 0.06174388900399208, -0.48961341381073, 0.16436946392059326, 0.585639476776123, -1.221198558807373, -2.463341236114502, 0.9100375175476074, 2.666260242462158, 1.2019038200378418, 0.9876737594604492, -0.5046825408935547, -0.010638384148478508, 0.988588809967041, -0.326276957988739, 0.3741534352302551, 2.1595065593719482, -0.3826020061969757, -0.48789042234420776, -0.7108191251754761, -0.7871112823486328, 0.6233690977096558, -1.088672161102295, -0.2648937404155731, 0.24825215339660645, -0.009703896939754486], [0.6559037566184998, -0.4268377125263214, 0.39305049180984497, -0.8625838756561279, 0.5453738570213318, 0.4008963406085968, -1.4819550514221191, -0.7084718346595764, -0.030900128185749054, 0.5364237427711487, 1.8340132236480713, 2.4345836639404297, 1.2938815355300903, 0.6063631176948547, 0.4748383164405823, -2.448216438293457, -0.5852175354957581, -0.7939383387565613, 1.234555721282959, -2.050693988800049, -0.656848132610321, 0.17230813205242157, -0.6876431703567505, -1.3405095338821411, 1.235229730606079, -0.27915358543395996, -0.6301853656768799, -0.48193520307540894, -1.8114526271820068, -0.6391791105270386, -0.6212707757949829, 1.4562431573867798, -0.8377465009689331, 0.37277671694755554, -1.1574833393096924, 0.052313193678855896, 1.2824108600616455, 0.6538106203079224, 1.8491778373718262, -0.03939221054315567, -0.2972443401813507, -0.9665132164955139, 0.1359180212020874, -1.0376261472702026, 0.28166863322257996, -0.1233510673046112, 0.6029323935508728, -2.6871728897094727, 2.065639019012451, 0.18536287546157837, -1.6511709690093994, -1.9261221885681152, 0.5689292550086975, 0.5313403606414795, 0.8625783920288086, 0.5859656929969788, 0.11954455077648163, -0.5622360706329346, 0.14438898861408234, -0.020707685500383377, -1.5768756866455078, 0.4191622734069824, -1.1989678144454956, 0.7022201418876648, -0.4113747179508209, -0.49929970502853394, 0.9830791354179382, -1.130050778388977, 1.736675500869751, 0.5607826709747314, 1.6317263841629028, 0.7882132530212402, 0.21928510069847107, 0.9553471207618713, 0.37344956398010254, -0.13827668130397797, 0.22643795609474182, 1.681378960609436, 0.49702566862106323, 0.28783532977104187, 1.0210003852844238, 1.770043969154358, -0.6858842372894287, 0.13423581421375275, 0.1168954148888588, 0.2335282415151596, 0.5280119776725769, -0.46859651803970337, -1.165941834449768, 0.4860724210739136, 1.246571660041809, -1.1976432800292969, -0.8238363265991211, 0.8685659766197205, 0.09715834259986877, -1.2526084184646606, -1.7865992784500122, 0.0023447589483112097, 0.47137877345085144, -1.6580195426940918, -1.0469051599502563, -0.0957634225487709, -0.18063059449195862, -1.0565992593765259, 0.4192942976951599, -2.170121908187866, 1.393223524093628, 0.3047308921813965, 1.0002373456954956, 1.4875683784484863, -1.4738492965698242, 0.059130363166332245, -0.9481194019317627, -1.5545597076416016, 0.28834399580955505, -0.09555614739656448, 0.3941015303134918, -0.13470692932605743, -0.4410698413848877, 1.9157766103744507, -0.4412045478820801, 1.5111234188079834, -2.0468783378601074, 0.015590139664709568, -0.04037958011031151, 0.2941659986972809, 0.0620553083717823, -1.012992262840271], [-0.0031451599206775427, 0.08223199844360352, 1.0358734130859375, -0.26109927892684937, -0.3401108980178833, -0.3908464014530182, 1.889465570449829, -1.4131778478622437, 1.4549005031585693, 0.7840468883514404, 1.4375029802322388, -0.5461190342903137, 0.804932177066803, -0.9714084267616272, 1.0486663579940796, 1.155489206314087, -0.22631461918354034, -3.175194263458252, -1.2217060327529907, 1.9578797817230225, 0.03846035152673721, 1.4918663501739502, -0.594150960445404, 0.029251649975776672, 1.1272250413894653, 1.6284757852554321, -0.7863754034042358, 0.8241734504699707, -1.0247480869293213, -0.5644903182983398, 0.36578479409217834, 2.1742708683013916, -0.5198559165000916, -1.1536122560501099, 1.785590648651123, -0.5644139647483826, 0.9460642337799072, 1.2229297161102295, 0.37328726053237915, 0.23345696926116943, -0.22234120965003967, -0.46969661116600037, -0.6405660510063171, -1.2932206392288208, 0.3720291256904602, -0.7166185975074768, 0.9042131900787354, 1.225283145904541, 1.8695809841156006, -0.014692376367747784, 0.09036017954349518, 0.5282673239707947, -0.8997334837913513, 1.4735702276229858, 0.01918351650238037, 0.34316515922546387, -1.0030910968780518, -0.0647997185587883, -1.0761350393295288, 1.2449898719787598, -1.1428396701812744, -1.1960136890411377, 0.3401263952255249, -0.4346303939819336, -0.9363052248954773, 0.7466704845428467, -0.39519166946411133, -1.1605443954467773, -1.336423397064209, -1.9113984107971191, 0.8727854490280151, 1.3384324312210083, -0.3648153841495514, 0.7014843821525574, -0.6900734305381775, 0.2204200178384781, -0.7566168308258057, 0.6860210299491882, -0.026880918070673943, -0.8346561789512634, -0.4777967929840088, -2.605442523956299, 1.9266588687896729, 0.19857698678970337, -1.4584619998931885, 0.11449350416660309, 0.887732207775116, 0.989472508430481, -0.49323952198028564, 1.0485821962356567, -1.5464049577713013, -0.9865034818649292, 0.020760977640748024, -1.938530683517456, -0.11782736331224442, 0.022173168137669563, 0.5147164463996887, 0.5233492851257324, 0.8062058091163635, -1.9478968381881714, 1.142939805984497, 0.9331157803535461, 0.2479175627231598, -0.9231756925582886, -0.3650491237640381, -0.2880319356918335, 1.291776180267334, 0.02876215986907482, 0.21359363198280334, -0.4895077645778656, 0.3304283022880554, -0.04588361829519272, -1.2884094715118408, 0.5453521609306335, -0.12638990581035614, 0.5608648657798767, -0.4771922826766968, 0.37385353446006775, 0.6241872906684875, 0.880945086479187, -0.13700281083583832, -0.6712579727172852, -0.9152384400367737, 0.6955183744430542, -0.124404676258564, -0.0768345296382904, -0.5550076365470886, -2.0861783027648926], [0.7958112359046936, -0.7997747659683228, 0.7359886169433594, -0.2513444423675537, -1.524164080619812, -0.5564390420913696, 0.3394278287887573, -2.420043706893921, 1.990836262702942, 1.6161019802093506, 0.2693740725517273, -0.251983106136322, 1.1690953969955444, 0.688943088054657, -0.5986066460609436, 0.7873536348342896, 0.2899291217327118, 0.4678974449634552, -0.39445826411247253, 2.142622947692871, 0.7684230208396912, -0.3965446352958679, -0.7678486704826355, 2.2940495014190674, 0.4610915184020996, -1.2723653316497803, -1.1102957725524902, -0.3469470739364624, -1.2604140043258667, -1.8566640615463257, -2.9250192642211914, 1.030051827430725, -0.7901737093925476, -0.3814235031604767, 0.19703823328018188, 0.30504587292671204, 0.27453941106796265, 0.5547336935997009, 1.2490594387054443, -0.1438155472278595, 0.42675232887268066, -2.484389305114746, 1.0028698444366455, 0.6461851000785828, 2.0532138347625732, -0.06951991468667984, -0.5667405724525452, -0.059755291789770126, -0.3960285186767578, 0.44668757915496826, 1.7170631885528564, -0.5107032060623169, 0.23416534066200256, -1.4317902326583862, 0.9354521632194519, -1.9900929927825928, 0.8172962069511414, -0.792061984539032, 0.07294544577598572, -2.005326986312866, -0.3178275525569916, -0.636457085609436, 0.4027453362941742, -1.8290364742279053, -2.0544514656066895, -0.37015119194984436, -0.9918738007545471, 3.124009132385254, -0.34467193484306335, -0.015339825302362442, 0.4970197379589081, 0.6497238874435425, -0.2631169259548187, 0.32258471846580505, -0.0486484169960022, -0.8129324913024902, -0.9284681081771851, -1.9989542961120605, 0.20731092989444733, -0.6877231597900391, 0.6674706935882568, -1.3634834289550781, -0.25492018461227417, 2.060357093811035, 0.014756161719560623, -1.80897855758667, -0.30662766098976135, 0.4662407338619232, 1.936745524406433, -0.9394984245300293, 1.0868687629699707, 0.33918482065200806, 0.6005898118019104, 0.7121475338935852, 1.5212589502334595, 1.312849760055542, 1.0315343141555786, -1.0063536167144775, -0.9110303521156311, 0.011521674692630768, 0.34875622391700745, -0.8466292023658752, 1.1178561449050903, 0.6952405571937561, -0.2764241695404053, -1.5785695314407349, 1.1827231645584106, -0.8602499961853027, 0.5387913584709167, 1.6116453409194946, 0.022895997390151024, -0.4192247986793518, -0.840294361114502, -1.4386987686157227, 0.39891451597213745, 2.3232827186584473, -0.4560023248195648, 0.48129773139953613, -1.0261353254318237, 2.025765895843506, 0.948205292224884, 0.02447197027504444, 0.1323169767856598, 0.45989230275154114, 1.2040963172912598, 1.054901123046875, 0.5936575531959534, 0.36125245690345764]], [[-1.2589435577392578, -0.19699649512767792, -0.398750901222229, -0.9587357044219971, -0.1271073967218399, -0.7481719255447388, 1.2984238862991333, 0.11412165313959122, -0.1567203849554062, -2.1687071323394775, 0.44232887029647827, -1.3100528717041016, -0.8188169002532959, -0.8261989951133728, -0.11409012973308563, 0.18933774530887604, -1.6589686870574951, 0.3808494210243225, 0.35054245591163635, 0.8028475046157837, -0.20031975209712982, 0.17589472234249115, -0.0023361460771411657, 0.37273433804512024, 1.0449035167694092, -1.516155481338501, -1.7917544841766357, -0.7676394581794739, 0.6822829842567444, 0.2370678037405014, -0.7805464267730713, -0.5445437431335449, -0.7170165777206421, 0.2960444390773773, 0.07527466118335724, -0.25847330689430237, -1.2332885265350342, -1.4950494766235352, 0.6928369998931885, -0.1113823652267456, 0.8621842861175537, -0.3484310209751129, -0.37328970432281494, -0.8541504740715027, 0.8504679203033447, -0.233438178896904, -0.008009371347725391, 0.22193096578121185, 0.0928807407617569, 1.2537431716918945, 1.0366606712341309, -0.7582732439041138, 0.1786165088415146, -0.8872256875038147, -0.42714476585388184, -1.4722250699996948, -0.21796824038028717, -0.3692649006843567, -1.0641051530838013, -1.8205431699752808, -0.4839494824409485, -0.2693183422088623, 1.819100022315979, 0.20571669936180115, 1.0568407773971558, -0.2586432993412018, 0.8315978646278381, 1.1964848041534424, -1.1192764043807983, 0.48539790511131287, 0.9453612565994263, 0.0680459514260292, 1.2251191139221191, 0.7247738838195801, -1.171426773071289, 0.5345793962478638, 0.5315229296684265, 0.003938966430723667, -1.107108235359192, -1.6518217325210571, 0.9989053606987, 2.3057820796966553, -0.4227147102355957, 0.6009110808372498, -1.1244733333587646, -0.7592560648918152, 1.4886847734451294, -0.48480772972106934, 0.024746114388108253, 0.4808177053928375, 1.3718968629837036, 0.6260755658149719, 0.8043240904808044, 0.9655159711837769, 0.5655470490455627, -0.038383759558200836, -0.23035882413387299, 0.3845987915992737, 0.5344808101654053, -0.020752709358930588, -0.29085466265678406, -0.14261619746685028, -1.6740115880966187, -0.16523095965385437, 0.39194366335868835, -1.4939558506011963, -0.11959809064865112, 0.6358680725097656, 0.3564881980419159, 0.3232024013996124, -0.6789329648017883, -0.7735521793365479, -1.0842152833938599, -0.6086491942405701, -0.09991699457168579, -0.7055332660675049, -0.7349187731742859, -0.7766923308372498, 0.47549140453338623, -2.1749017238616943, 1.0000123977661133, 0.700461745262146, 0.1521635800600052, 1.140546202659607, -0.6850703358650208, -0.7992282509803772, 0.09156778454780579, 0.8731613755226135], [1.2339413166046143, -1.4654085636138916, 1.1887322664260864, -1.0095607042312622, -0.3379390239715576, 0.9273539185523987, 0.566438615322113, 1.0878198146820068, 0.36965861916542053, 0.0939774289727211, 0.6979000568389893, -1.1226236820220947, -0.5379952192306519, 0.3904878497123718, 1.3194451332092285, -0.6050043106079102, -0.7194542288780212, 0.12762607634067535, 1.604215383529663, 1.5995503664016724, 0.4599791169166565, -1.1899977922439575, -0.019257664680480957, 0.6868289113044739, -0.5029391646385193, -0.7104091048240662, -1.0018935203552246, -1.8473793268203735, -0.9920760989189148, 0.4900748133659363, -0.2840851843357086, 0.07092025876045227, -0.22861024737358093, 0.33353087306022644, -0.37598782777786255, -0.6783225536346436, 0.07791285961866379, -0.5655706524848938, -0.3689725995063782, 0.22498150169849396, -1.3233352899551392, 1.0726715326309204, -0.31446343660354614, 0.4761717617511749, 0.9447047710418701, 0.45266881585121155, 0.7295567393302917, -0.6909409165382385, 0.7973021268844604, -0.6968880891799927, 1.3417458534240723, -0.29325130581855774, -0.07884206622838974, -0.37681540846824646, 0.19886581599712372, 0.5296636819839478, -0.7023754119873047, -0.43062371015548706, -0.01437170896679163, -0.6289149522781372, 1.059820532798767, 0.8221667408943176, 0.8424498438835144, -0.16770750284194946, -0.37233400344848633, -1.0413509607315063, 0.7492767572402954, 0.46709999442100525, -0.3028395473957062, -0.7673400640487671, 1.1127078533172607, -0.9732344150543213, 1.4186972379684448, 0.6226288080215454, 0.43010878562927246, -0.5424754023551941, 0.8145977258682251, 0.05943011865019798, -0.32745087146759033, -0.909474790096283, -0.043509915471076965, -0.6981567144393921, -0.6725155711174011, 0.295147180557251, 1.399468183517456, -1.555018663406372, -2.132687568664551, 1.3236075639724731, 1.2001800537109375, -0.3086149990558624, 0.5916662812232971, 0.11813399940729141, -0.0037685621064156294, 0.045769836753606796, 1.5439510345458984, -0.03888928145170212, -0.8977900147438049, 0.50658118724823, 0.7164574861526489, -0.56959068775177, 1.4945836067199707, -0.2120957225561142, 1.1782792806625366, 0.7703239321708679, 0.07211333513259888, 1.1640278100967407, 0.2766512632369995, 0.8885301947593689, 0.30614539980888367, -0.8803881406784058, -1.257310390472412, 0.11984928697347641, -0.08755193650722504, -0.38944244384765625, -1.4617382287979126, 0.14533449709415436, -0.13385902345180511, 0.8993995785713196, 0.2775101065635681, -1.0047701597213745, -1.104314923286438, 0.1057816818356514, 1.7351139783859253, 0.11978071928024292, 0.9786056280136108, 0.6658205986022949, 0.0696958377957344, 1.0251374244689941], [-0.31628021597862244, 0.08931713551282883, -0.4974367916584015, -1.8747179508209229, 0.4729136526584625, -1.7591785192489624, -0.11747387051582336, 0.4592646658420563, 0.0486791618168354, 1.3835852146148682, 1.664156436920166, -0.25210487842559814, -0.7145796418190002, 1.393014907836914, -0.9832071661949158, -1.4435943365097046, 0.775087296962738, 1.4212391376495361, -0.7526445388793945, 1.2612111568450928, 1.171151876449585, 0.9296427369117737, -0.9503833651542664, -0.5069589614868164, -0.31064873933792114, -0.7671958804130554, -0.8626927137374878, 0.2596735656261444, 0.4459022283554077, 0.25537341833114624, -2.17045259475708, 0.633768618106842, -0.41923007369041443, -1.4358826875686646, 0.2754841446876526, -1.09380304813385, -1.0480989217758179, 1.6787633895874023, 0.8066596984863281, -0.5623830556869507, 0.42689836025238037, 0.7901906967163086, -0.164365753531456, 1.3394941091537476, -0.626461923122406, -1.1201311349868774, 0.3264836370944977, 0.5057755708694458, 0.2160889208316803, 0.38912343978881836, -0.9344250559806824, 0.06591479480266571, -1.402959942817688, -0.033459629863500595, -1.051672101020813, 1.36225163936615, 0.20318004488945007, -0.9206568598747253, -0.1548401564359665, 0.29525625705718994, 0.8045347929000854, -0.47328999638557434, -0.01229005679488182, 1.2116223573684692, -1.0964233875274658, 0.4141461253166199, 0.7612835764884949, 0.8390480875968933, -0.1210474893450737, -0.7527174949645996, 0.8345014452934265, 0.4123820960521698, -0.4702219069004059, 1.099543809890747, -0.8743793368339539, 0.23251040279865265, -1.592147946357727, -0.8578248620033264, -0.727105438709259, 0.4377683997154236, 1.7510966062545776, 0.6892760992050171, 0.29286524653434753, 2.007040500640869, 1.201265573501587, -0.5953064560890198, 0.40869152545928955, 0.6605613231658936, 0.08632925897836685, 0.2977343499660492, -0.6304131150245667, -0.7485027313232422, -0.3547171950340271, 1.3353323936462402, 1.0522559881210327, 0.3596850037574768, 0.38074252009391785, -1.7069497108459473, -1.5944737195968628, -1.682751178741455, -1.6174763441085815, 0.6946303844451904, -0.18046188354492188, -0.9985820651054382, 1.2351281642913818, 0.4459129869937897, 1.822957992553711, -2.0570263862609863, -0.7100716829299927, -1.743041753768921, -1.4699679613113403, 0.6577796936035156, -1.0857735872268677, 1.2737171649932861, 0.7915096879005432, 0.4336259365081787, 0.06704524904489517, 1.544189691543579, 0.32907968759536743, 0.41575804352760315, 1.003401517868042, 1.2113256454467773, 1.316055417060852, -0.7369652986526489, -0.6967781782150269, -0.493590384721756, 0.219572514295578, -0.04556716978549957], [2.134887933731079, 1.7481762170791626, -1.157083511352539, 0.5227275490760803, 0.23331062495708466, -0.7004566788673401, 0.4400951862335205, 1.1103084087371826, -0.21283291280269623, -0.04650122672319412, -1.1327776908874512, -1.196610927581787, -0.8551636338233948, 0.099252849817276, 0.19544635713100433, -1.357153296470642, 0.9935499429702759, 2.9445321559906006, -0.26113423705101013, -0.6280916333198547, 0.4309006333351135, -1.930724859237671, 0.05048755556344986, -0.3916396498680115, -1.5619832277297974, 0.1556060016155243, 2.1065988540649414, 0.6012328267097473, -1.2583215236663818, -0.31419453024864197, 1.043544888496399, -0.3851580023765564, -1.0424702167510986, -0.32824525237083435, -0.19137296080589294, -1.5910134315490723, -0.03083980269730091, -1.9415006637573242, -0.05084050074219704, -1.0229612588882446, 0.10241146385669708, -0.05402239039540291, -0.6366304159164429, 0.8142478466033936, 0.4682080149650574, -2.6010730266571045, -0.24485638737678528, 0.02436547912657261, 0.03055541217327118, 0.43272078037261963, -1.3347153663635254, -1.0763769149780273, 0.5988278388977051, 0.5772961378097534, 0.3463962972164154, 1.5724694728851318, -0.9241601228713989, -0.04786065220832825, 0.663076639175415, 1.0928508043289185, -0.4854474663734436, 0.6422044038772583, -0.8099012970924377, 0.20008043944835663, -0.6030075550079346, -0.11302492022514343, -1.1178200244903564, -0.9490301609039307, -1.6220052242279053, -0.5778638124465942, 0.0208777766674757, -0.3951593339443207, 0.6082813143730164, 0.38085034489631653, 1.0370017290115356, 0.4799509048461914, -0.9283266663551331, 0.09818068891763687, -0.24699446558952332, 0.9600831270217896, -0.7090691328048706, -0.6282224059104919, -1.3057864904403687, -1.0073859691619873, -0.5203007459640503, -0.6758904457092285, 1.1794816255569458, 1.519784688949585, 0.4897006154060364, -0.9480928182601929, -1.548204779624939, -0.42118746042251587, 1.5827628374099731, 0.37386801838874817, 0.1747925579547882, 0.8563616871833801, -0.7524288892745972, 0.04748687148094177, 0.5435047745704651, 1.1703381538391113, -0.014162297360599041, -0.6470508575439453, 1.2886855602264404, 2.6851248741149902, 0.6546337008476257, -2.3794214725494385, 2.211298942565918, -0.8010909557342529, 0.9493356943130493, 0.21615152060985565, 0.12261901795864105, 0.19061753153800964, -1.3136675357818604, 1.2226969003677368, -0.8451312780380249, 0.828135073184967, 0.05488881096243858, 0.7167903184890747, 0.08486396819353104, -1.5072938203811646, 1.5409045219421387, 0.7660509943962097, 0.4483063220977783, 0.6016902327537537, -1.5584619045257568, 0.9736907482147217, -0.8054298758506775, -0.5919888019561768], [0.018010899424552917, -0.34224390983581543, 0.5328109860420227, -1.4363195896148682, 2.309537410736084, 0.46183907985687256, 0.6113755702972412, 1.2918552160263062, 1.5878546237945557, 0.37823835015296936, 0.5771960020065308, -0.727795422077179, 1.4842637777328491, 0.2704964578151703, -1.9460676908493042, -0.5494303107261658, -1.5586278438568115, -2.1593728065490723, 1.077150821685791, -0.3955468535423279, -0.46992695331573486, 0.36774152517318726, -1.790350079536438, -0.2991555631160736, -0.39378228783607483, 0.6527962684631348, 0.282532662153244, -1.4914486408233643, 0.3036338686943054, -0.9401665329933167, 0.20087316632270813, -1.3352866172790527, 0.15404638648033142, -0.13381902873516083, 0.15988720953464508, -0.771580159664154, 1.3452847003936768, -0.6010440587997437, -0.7823401093482971, 1.2839384078979492, 0.3914244472980499, 0.9449812769889832, 0.06404981017112732, -1.5949829816818237, -1.3635988235473633, 0.10366271436214447, -0.837889552116394, -2.8040196895599365, -2.4373421669006348, -1.8691514730453491, -1.5531635284423828, -1.9103931188583374, 0.6393008232116699, 0.28218138217926025, 0.1792791187763214, 0.910283088684082, 0.33027008175849915, -0.11993958801031113, 0.6370536088943481, -0.2575605511665344, -1.784145474433899, -0.11641636490821838, 0.0612495020031929, -1.2815821170806885, -0.715828537940979, 0.21556688845157623, -0.6417421102523804, 0.9016455411911011, -1.7979146242141724, -0.7642856240272522, 0.42314648628234863, -2.5392282009124756, -0.27432504296302795, -0.10652422159910202, -1.629648208618164, 0.47570401430130005, -0.06905718147754669, -0.3889872133731842, -0.6779941320419312, 0.5496000051498413, -0.3419061005115509, -0.27458712458610535, -0.026922304183244705, 0.7576997876167297, 1.9106881618499756, -0.7589210271835327, -0.8352739810943604, 0.46210500597953796, 0.8982295393943787, 0.5257142782211304, 1.5379270315170288, -0.2660980522632599, 0.05791591852903366, 1.063094139099121, -0.4883732199668884, -0.8685011267662048, 0.029561365023255348, 0.41821783781051636, 3.572566270828247, 0.3041384816169739, 1.1097184419631958, 0.7196033596992493, 1.1208906173706055, 0.29603591561317444, -0.7728869915008545, -0.1404029130935669, -2.6502435207366943, 1.4208959341049194, 0.19281432032585144, 0.16720078885555267, -0.4062815010547638, 0.3666817545890808, -0.9008250832557678, -0.0482209138572216, -0.6472979784011841, 0.2005627602338791, 0.6387302279472351, -0.5625503063201904, -1.381476640701294, 0.957680881023407, 0.11329075694084167, -0.6305372714996338, -0.6431508660316467, -2.480588912963867, -0.32820427417755127, -0.46346616744995117, 0.5170614123344421, 1.8122652769088745], [-1.1528499126434326, -1.411811351776123, 1.2093532085418701, -0.6741794347763062, -0.7495042085647583, -0.33908435702323914, -0.12282104790210724, 1.6816679239273071, 0.6849623322486877, 1.013077974319458, -0.8141584396362305, -0.05298713594675064, -1.8468223810195923, 1.7160844802856445, 1.3240232467651367, -1.2677069902420044, -0.12484761327505112, -0.9102590680122375, 1.8038101196289062, 2.2592029571533203, -1.2498403787612915, -0.5801215767860413, 1.2999402284622192, -0.3564779460430145, -0.8532077074050903, -3.320289373397827, -0.9128968119621277, 1.0907610654830933, -0.3069359064102173, 0.3379422724246979, -0.41872721910476685, 0.7341175675392151, 1.4564775228500366, 0.7635170817375183, -1.4791597127914429, -0.13923871517181396, -1.6517659425735474, -0.4455457031726837, -1.0062462091445923, -1.1347626447677612, 1.1917593479156494, 0.49531370401382446, 1.540221929550171, 0.1495605856180191, -1.3192647695541382, 1.3340375423431396, -0.30409887433052063, -0.7061998844146729, 1.343515157699585, -0.08195508271455765, 0.11391457915306091, -1.7381311655044556, 0.1950892210006714, -0.38246604800224304, -2.269725799560547, 0.5044454336166382, -0.03698413074016571, -0.012607095763087273, -0.06558732688426971, 0.18557697534561157, -1.8475031852722168, -0.4719860255718231, -0.786945641040802, -0.537203311920166, -0.4877378046512604, 0.14465469121932983, 0.5566456913948059, 0.3329131603240967, 2.015498161315918, 2.4096925258636475, 0.8274649977684021, 1.0939092636108398, -0.1679486334323883, 0.13960422575473785, -0.5581537485122681, 0.036178432404994965, -1.036972165107727, 0.04182463511824608, 1.5597018003463745, 0.7462849020957947, 1.5629714727401733, -0.1175791472196579, -0.1896049827337265, -0.6457514762878418, -1.304153323173523, 0.43578195571899414, -1.918274998664856, -1.2311745882034302, 0.9927929043769836, -0.2665417492389679, 0.3850838541984558, 1.0214455127716064, 0.8848719596862793, -0.23022392392158508, 0.41492730379104614, -1.5819517374038696, -0.7756546139717102, 1.9544047117233276, -0.1939634084701538, -1.2075318098068237, -0.8704164028167725, -1.7963072061538696, -0.5692726373672485, 2.456510066986084, 1.111270785331726, -0.9030163288116455, 0.03651324287056923, 2.12591552734375, 0.37599098682403564, 2.038048028945923, -0.37770944833755493, -1.0919543504714966, -0.6845356822013855, 1.0534896850585938, -1.6750974655151367, 1.304257869720459, 0.7385644316673279, 0.3547057807445526, -0.26225975155830383, 0.600619375705719, 0.775225043296814, 1.38548743724823, 0.7293084263801575, 0.9337021708488464, 1.658394694328308, -1.3823732137680054, 2.771230936050415, -0.6914165019989014], [-0.12681706249713898, 0.7279068231582642, -0.6914908289909363, -0.46473440527915955, -0.9764381051063538, -0.5715213418006897, -2.689220428466797, -0.08710449188947678, -0.5954190492630005, 0.5459666848182678, -0.25009945034980774, -2.1983144283294678, 2.3635268211364746, -0.2583036720752716, 0.9730275273323059, 0.019387638196349144, 0.2892420291900635, -1.6086400747299194, -0.24113453924655914, 1.3626080751419067, 0.3289034068584442, -0.9624720215797424, -0.2938535809516907, -0.798581600189209, -0.5127507448196411, 0.5186240077018738, 2.068978786468506, 0.2615514099597931, -0.48537346720695496, -0.1777779757976532, 0.4363507628440857, -0.6266314387321472, -1.4308466911315918, -0.3453153967857361, 0.29413166642189026, -0.7165011763572693, 0.2584981322288513, 2.028459310531616, -1.0152149200439453, -0.47348129749298096, 0.6263444423675537, 1.709251880645752, 2.0121536254882812, -1.0108981132507324, 0.8951091170310974, -0.17845045030117035, -0.07555214315652847, 0.4839267134666443, -2.076148509979248, 0.09071176499128342, -1.2679097652435303, 0.3072340488433838, -0.8866868615150452, -0.48871901631355286, 0.3380625545978546, 0.2147504836320877, 0.18423223495483398, 0.9387712478637695, 1.3375760316848755, 1.1890393495559692, -0.6310360431671143, 0.7661644220352173, 0.0988561138510704, 1.1350514888763428, -2.0322043895721436, -0.441508948802948, 0.7127591967582703, 1.2636638879776, 1.9791537523269653, 0.11232469975948334, 1.4787602424621582, -1.124474287033081, 0.2617306411266327, -0.4637377858161926, -0.9481508731842041, -0.050383489578962326, -0.3940250277519226, 0.90362948179245, -1.6354933977127075, 0.4970889091491699, 1.9000065326690674, -0.9906184077262878, -0.30533865094184875, -0.7834120392799377, 1.3031941652297974, -0.672453761100769, -0.17360371351242065, 1.719253420829773, 0.9742052555084229, -0.7383782863616943, -0.03758695349097252, 0.8118271827697754, -0.7068163156509399, -1.2412803173065186, -0.4533597230911255, 0.7208154797554016, -1.287535548210144, -0.4620665907859802, -2.3369827270507812, -0.21601402759552002, -1.4691882133483887, -0.019563427194952965, 0.915986955165863, -0.2818189561367035, -0.5720125436782837, 1.1279405355453491, -0.9720061421394348, 0.4726954996585846, -1.8857427835464478, -0.9662348031997681, -0.2923581302165985, -0.4423416554927826, -0.4033842980861664, -2.1095523834228516, 0.42230474948883057, 0.13189291954040527, 1.2925612926483154, 0.6062468886375427, -1.4743866920471191, 1.0801379680633545, 0.7481030821800232, 1.2272087335586548, -0.6864854693412781, -1.4124386310577393, 0.21854408085346222, -0.16133631765842438, -0.050156861543655396, -1.5606293678283691], [-1.8209178447723389, -1.50877046585083, 1.3131619691848755, 0.4631865918636322, -0.9977114796638489, 2.519200563430786, 0.14373888075351715, 0.13752244412899017, -1.7218588590621948, -1.1948778629302979, -0.957884669303894, -1.2053805589675903, -0.6215386986732483, 1.190258502960205, 0.790820300579071, -0.9955974221229553, 0.4070607125759125, 0.1295388787984848, 0.7171757221221924, -0.25520437955856323, 1.742403507232666, -0.537855327129364, -0.6965174674987793, 1.0151935815811157, -0.6987254023551941, 0.6257919073104858, -0.8077862858772278, -0.2745034098625183, 0.3746844232082367, -0.49037837982177734, -0.7003166079521179, -2.0194287300109863, 1.0852165222167969, -0.7954407930374146, -1.1550171375274658, -0.2924720048904419, 0.2015388309955597, 0.03876276686787605, 1.314696192741394, 1.6502363681793213, -2.176787853240967, -0.1978577971458435, -0.4202231466770172, 0.03239153325557709, -1.3407105207443237, 0.4342557191848755, 1.338306188583374, -0.3864499032497406, -0.2055668830871582, 0.3588240444660187, -0.20184564590454102, -0.12537835538387299, -1.704649567604065, 1.0950013399124146, 0.921394407749176, 0.12447631359100342, -0.12706759572029114, -1.7899808883666992, -2.804837942123413, 0.058100685477256775, -1.5758917331695557, -0.7806314826011658, 0.1251048594713211, 1.558593988418579, -1.6055738925933838, -0.009492514654994011, 0.8934255838394165, 3.8382394313812256, -1.0710068941116333, -0.3562861680984497, -0.8958008885383606, 1.0724678039550781, -0.7980605363845825, -0.6798520088195801, 0.06692753732204437, -0.501853883266449, -0.6013516783714294, -1.023325800895691, -1.7933751344680786, -0.5657215118408203, 0.39150407910346985, -1.5190613269805908, -0.25377076864242554, -0.5238391757011414, 0.03726811334490776, 0.9964103698730469, -0.5977025032043457, 1.2664226293563843, 0.02512321248650551, -1.6846439838409424, 0.47385644912719727, -0.9645575881004333, -0.4604126214981079, 1.836955189704895, 0.2544618844985962, -1.3135613203048706, 0.42586377263069153, 2.1844491958618164, 0.807424783706665, -1.208022952079773, -0.9719500541687012, 0.6414309144020081, -1.7691744565963745, 1.6533770561218262, 0.39747491478919983, 2.806809663772583, 0.962909996509552, -2.0396976470947266, -0.06315532326698303, 0.9271148443222046, -0.6689480543136597, -0.6230803728103638, 0.14749784767627716, -0.5268746614456177, -0.583829402923584, -1.5582942962646484, 1.314744234085083, -0.1195530891418457, -0.828899085521698, -1.46846604347229, -0.4939177930355072, -0.7406464219093323, 0.726701021194458, 0.03508612886071205, -0.910507321357727, -0.09061520546674728, 1.8571773767471313, 1.7535845041275024], [-0.6917139291763306, -1.6546765565872192, 0.04949484393000603, 1.0612432956695557, 1.9088910818099976, -0.6584452390670776, -0.22628305852413177, -0.03958745300769806, 1.467577338218689, -1.3817843198776245, -0.42733657360076904, 0.6598827242851257, -0.7489652037620544, -0.6431863307952881, -0.14791424572467804, -0.3787541687488556, -0.33384814858436584, -1.002510666847229, -0.005065153352916241, -0.43954038619995117, -0.05140930041670799, -0.023653047159314156, -0.028647001832723618, -0.37944841384887695, 0.36907434463500977, 0.4875796437263489, 1.6235992908477783, 0.9167732000350952, -2.7588987350463867, -1.6602662801742554, 0.8710394501686096, 2.3470993041992188, 0.01571086049079895, 0.7656692862510681, -1.6262564659118652, 0.6331489682197571, 0.004957945551723242, 0.7538238763809204, -0.03478005900979042, -0.2785907983779907, -0.3712213933467865, -0.7110332250595093, -0.7501758933067322, -0.0534224808216095, 0.49052664637565613, 2.5298190116882324, 0.2971828281879425, 0.8976858258247375, 1.2540432214736938, -1.0191905498504639, 0.3752223551273346, 0.38560202717781067, -2.4994170665740967, 2.598726749420166, -2.5117273330688477, -0.5583332180976868, -1.6901586055755615, 0.6752133369445801, 0.928987443447113, 0.1919766366481781, 1.7097054719924927, 0.5299773812294006, 0.6946293711662292, -1.8636291027069092, 0.2837240397930145, 0.7522716522216797, -0.9410545229911804, -0.538766622543335, -0.10934364795684814, -0.5194687247276306, 1.0153828859329224, -0.24953287839889526, 0.21523864567279816, 0.560248851776123, -0.04352631792426109, 0.7297568321228027, 0.7823652625083923, -0.6961772441864014, 0.9389952421188354, -0.17070718109607697, 0.631708562374115, 0.12379290163516998, -1.0583114624023438, -0.1281697303056717, -0.38964083790779114, -0.687044084072113, -0.9611250162124634, 1.3273193836212158, 0.8175681233406067, 0.8750435709953308, 0.32711565494537354, -0.16933801770210266, -0.3312912583351135, -0.45618125796318054, 1.5054253339767456, -0.48036545515060425, -0.7746046781539917, -1.2143137454986572, 0.3574486970901489, -0.9791609048843384, -1.6126512289047241, 0.046639375388622284, -1.0851057767868042, -0.641559898853302, 0.04314907267689705, 1.0549286603927612, 0.7001598477363586, -0.33169299364089966, -0.9882423281669617, -2.622713565826416, -0.18685726821422577, 0.3989599943161011, 0.23757559061050415, 0.06192982196807861, 0.09809314459562302, 0.37324216961860657, -0.24754808843135834, -0.5070050358772278, -0.8607664108276367, -0.6294440627098083, 0.02772885374724865, 0.6739934086799622, -1.0750523805618286, -1.2967760562896729, -0.08509865403175354, -1.1344376802444458, -0.5103552341461182, -1.0209630727767944], [0.4559413492679596, -0.9838285446166992, 0.1753593236207962, 0.2900483310222626, -1.6159205436706543, -0.11357876658439636, 0.3732747733592987, 1.0587830543518066, -1.5802083015441895, 0.6179983615875244, 0.04440653696656227, -1.36572265625, 0.1961621791124344, 1.3925198316574097, 1.5476386547088623, 1.9967248439788818, -0.16496479511260986, 0.8619353771209717, -0.29698750376701355, 1.0878558158874512, -0.5265936851501465, -0.07764964550733566, 0.5413728952407837, 1.6240439414978027, 0.10295998305082321, -0.7606860399246216, -0.27438783645629883, 0.16469688713550568, 1.4108115434646606, -1.3483434915542603, 0.8200547695159912, 0.5481754541397095, -0.46230193972587585, 0.2999640703201294, 0.679972767829895, -0.9351767897605896, -0.5164346098899841, 0.03972591459751129, -0.2461184412240982, -0.11764821410179138, 0.41995999217033386, 0.38147658109664917, 0.5654390454292297, 0.14591148495674133, 0.43269044160842896, -2.0106964111328125, -0.7075744867324829, 0.5503678917884827, -0.9975140690803528, -0.5024551749229431, 0.3944331705570221, -1.846573829650879, -0.1435314118862152, -0.5971802473068237, 0.9560595750808716, 0.3676075041294098, -0.6512596607208252, 1.0463387966156006, -0.173463374376297, 0.05644708871841431, -0.9586559534072876, -0.05989587679505348, 0.3548028767108917, 2.009955883026123, 0.23583155870437622, -0.7916621565818787, -1.1256515979766846, -0.450040340423584, -0.450611412525177, -0.38503655791282654, -0.4970298409461975, 0.64311683177948, 0.5857415199279785, -0.41219326853752136, 0.7262128591537476, -0.3046438992023468, -0.3523974120616913, 0.7737962603569031, -0.42824044823646545, -0.9267284274101257, 0.19597232341766357, 0.22333796322345734, -0.9649034738540649, 0.9690079689025879, 1.4346182346343994, 0.7580299377441406, 1.0453541278839111, 0.9826677441596985, 0.10192494839429855, 0.009440433233976364, -1.9276888370513916, -3.278179883956909, -0.12839248776435852, -0.8153926134109497, 0.7046979665756226, 0.21564064919948578, -0.19123254716396332, 0.2952454686164856, 1.5132914781570435, -0.2559509575366974, 0.1917746365070343, 0.6561054587364197, 0.11010868847370148, -1.4151203632354736, 0.5853778123855591, 1.7411924600601196, -1.0563571453094482, 0.42542770504951477, -0.2639341652393341, 0.20257608592510223, -2.356050491333008, 0.897567093372345, -0.8319637179374695, 0.2197471708059311, -0.6041406393051147, 0.705793023109436, -0.02793513797223568, -1.0525767803192139, -0.9297210574150085, -2.360827922821045, 0.056234173476696014, 0.3704947531223297, 0.38860660791397095, -0.5559933185577393, -0.22395917773246765, 0.6931819915771484, 1.5691359043121338, -0.43105950951576233], [0.45888185501098633, -0.9166057705879211, -0.259651780128479, -0.9186668992042542, 1.645545482635498, 0.6624637246131897, 0.07414248585700989, -1.0948010683059692, 0.31406348943710327, -0.2242925465106964, 0.6515798568725586, 0.15206493437290192, -0.12077146768569946, 0.18132880330085754, 0.7218031883239746, 0.1435346007347107, -0.7147091031074524, -1.3775110244750977, 0.36329278349876404, -0.8149431943893433, -1.3519266843795776, 1.199825644493103, -1.160402774810791, 0.38629207015037537, 0.996593713760376, -0.09268838912248611, -1.2147425413131714, -0.5396618843078613, -0.1575428545475006, 2.1013917922973633, -1.2144464254379272, 0.1855410933494568, -0.19266727566719055, -0.7671384811401367, -0.31558719277381897, -0.5311905741691589, -0.686180055141449, 0.3338153660297394, -0.3357202708721161, -0.5566315054893494, -0.2701888084411621, 1.9316761493682861, 1.8231807947158813, 1.3106039762496948, -2.0279974937438965, 0.32775866985321045, -0.6215399503707886, 0.7459283471107483, -1.5306816101074219, -0.6703010201454163, -0.47002550959587097, 0.3082987368106842, 1.0773403644561768, -0.593979001045227, -0.43757933378219604, 1.238554835319519, 0.3703315556049347, 0.9981800317764282, -0.41405054926872253, 0.36470845341682434, -0.17747358977794647, -0.6471892595291138, 2.9992945194244385, -0.2269740253686905, 1.8187470436096191, 0.026279177516698837, -0.13257184624671936, -0.7470633387565613, -0.403065025806427, -0.5580453276634216, 2.8212270736694336, -1.1568750143051147, 0.4713656008243561, 0.977130115032196, 0.8465715050697327, 2.485597610473633, 1.5438603162765503, 0.8165083527565002, 1.581009864807129, 0.3573165833950043, 1.2786017656326294, 0.6150640845298767, -1.7141475677490234, -0.4977400004863739, 1.7888789176940918, -0.3140660226345062, -0.2130449265241623, 0.5167763233184814, -2.272049903869629, -0.5615922808647156, 0.08180128037929535, -0.3548131287097931, 0.4437432289123535, -0.48789259791374207, 0.7265754342079163, -0.07681101560592651, -1.4760229587554932, 0.7313848733901978, -1.2528187036514282, 1.621501088142395, 0.5067312717437744, 0.19228523969650269, 0.2393294721841812, -0.15045545995235443, 0.817139744758606, -2.02463698387146, -0.3222767114639282, -1.1466279029846191, -2.152791976928711, -1.6526620388031006, 0.5496074557304382, 0.03182917460799217, -0.26171159744262695, -0.4921361207962036, -0.9483136534690857, 0.09159788489341736, -0.22640635073184967, 1.02601158618927, -0.10358280688524246, -1.017577052116394, -0.7866242527961731, -0.011473606340587139, 0.1643245965242386, -0.6756913661956787, -0.22607757151126862, 1.2080588340759277, -0.7666933536529541, -0.796652615070343], [1.970449447631836, 2.17539381980896, -1.1391338109970093, 0.10294840484857559, 0.11870986968278885, 0.3708880543708801, 0.47176313400268555, -0.9622405767440796, 0.72599196434021, 0.22009502351284027, -1.6454890966415405, -0.5944589972496033, -0.23865772783756256, 0.6786492466926575, -1.5075194835662842, 0.16812865436077118, 1.8316859006881714, 0.34032562375068665, 1.6107807159423828, 0.262426495552063, 0.35860249400138855, 0.7954992055892944, 0.6742936968803406, 0.1451890617609024, -1.6152572631835938, 0.40294837951660156, -0.17514662444591522, -0.2608078718185425, -0.8189807534217834, 0.6707956790924072, 1.3759377002716064, -1.0625648498535156, 0.8711115121841431, -0.6539096236228943, 0.6995909810066223, 0.9359886050224304, 0.09579908847808838, -1.5679923295974731, -1.378150463104248, -0.3294970393180847, -1.8856498003005981, 0.5320886373519897, -0.4023624360561371, 0.3569641709327698, -0.6441870331764221, -0.6352744102478027, 0.11626628041267395, 2.0653862953186035, -1.1561009883880615, 0.06874645501375198, -1.0641412734985352, -1.9078750610351562, -0.20711907744407654, 0.9158540964126587, 0.13616108894348145, 0.7234067320823669, -0.45379337668418884, -0.9771726131439209, -0.7316080331802368, 0.24369032680988312, 0.17056109011173248, 1.0590397119522095, 1.0410125255584717, 0.3315708637237549, -0.8633633255958557, -1.489214301109314, -0.8427106142044067, -1.0441707372665405, 0.9076347947120667, 0.19226357340812683, -0.3920289874076843, -0.9240773916244507, -0.060506921261548996, -0.7664979696273804, 0.177830770611763, 0.5135478377342224, -0.15051774680614471, -0.09741484373807907, 0.7750514149665833, -0.22975370287895203, 1.3897929191589355, 1.022821068763733, 0.4275059998035431, 0.8314017653465271, -1.3787450790405273, 0.6283820271492004, -0.9907047152519226, -0.7067237496376038, 0.34260979294776917, -1.778470754623413, -1.133459448814392, -1.5732280015945435, -1.207309365272522, -1.3218914270401, 0.5403625965118408, 0.32105374336242676, 0.3543836176395416, -0.9775493741035461, 0.3865041136741638, 1.436274528503418, -0.29885098338127136, 0.34838688373565674, 1.1890757083892822, 0.7457231283187866, -1.2577577829360962, -1.6057380437850952, 0.26070573925971985, -1.8652664422988892, -0.9100098609924316, 0.3023446500301361, 0.13054929673671722, -1.6744636297225952, 1.0398660898208618, 1.1049344539642334, -1.7016319036483765, 0.393553763628006, -2.8301477432250977, -1.6561988592147827, 1.4793999195098877, -1.3794443607330322, 1.185109257698059, -1.4539045095443726, -0.36421599984169006, 0.3420179784297943, -1.1301008462905884, -0.7503960132598877, -0.44685977697372437, -0.22608192265033722], [-0.5656660795211792, -1.394594430923462, 0.2017362415790558, -2.1308159828186035, 2.448660135269165, 0.7551138997077942, -1.000515103340149, 0.31187474727630615, -0.856939435005188, 0.30452924966812134, 0.6453704237937927, 0.39695602655410767, -1.0749287605285645, 0.3170643746852875, 1.8662863969802856, 1.047831416130066, -1.014852523803711, 0.9643197655677795, 0.04031520336866379, -0.05996564030647278, -0.23990637063980103, 1.3280391693115234, -0.6062027812004089, 0.647763192653656, -0.9665992259979248, 0.5557414889335632, 2.1431691646575928, 0.4145876169204712, -0.7870485186576843, 1.0580143928527832, 1.6975786685943604, 1.0848923921585083, -0.7950186133384705, 0.4630567133426666, -0.20404669642448425, -0.21898119151592255, -1.1433024406433105, 2.1051137447357178, -1.7087700366973877, -0.11687624454498291, 1.637935996055603, -1.5722140073776245, 1.9137862920761108, -1.0065327882766724, -0.48920929431915283, -2.5551958084106445, 0.08162366598844528, -1.1681265830993652, -0.2409273236989975, -0.22078004479408264, -0.6567196846008301, 0.5147741436958313, 0.7973721623420715, -0.3781830370426178, -0.7865314483642578, 0.16806593537330627, 0.8268371820449829, -0.09149302542209625, 1.0152209997177124, 1.9789891242980957, -0.6903791427612305, -0.44334283471107483, -0.2564488351345062, 1.1934765577316284, 1.523734211921692, -2.069441318511963, 0.638046145439148, -0.4856279194355011, -0.14718331396579742, -1.2073785066604614, 0.14913272857666016, 1.9782824516296387, 0.041801560670137405, -1.2955429553985596, -1.2173560857772827, 0.8999595642089844, 0.44923627376556396, -1.5857131481170654, -0.3470585346221924, -0.16964171826839447, 0.16563715040683746, 0.049123093485832214, 0.5099523663520813, -0.1627485752105713, 1.3957217931747437, 2.0202724933624268, -1.5553308725357056, 0.36259961128234863, -1.2213562726974487, -1.0902959108352661, -1.3531967401504517, 0.6766093969345093, -0.099940225481987, -0.2952222228050232, -0.3072609007358551, -1.3466050624847412, 0.1644100695848465, -0.5538284778594971, -1.4918969869613647, 1.7173774242401123, -1.4154590368270874, 1.204453945159912, -1.168290376663208, -0.0017897390061989427, -0.7523678541183472, -0.61717689037323, -0.5234233140945435, -1.065686821937561, 0.14455926418304443, -0.9473454356193542, -1.4342693090438843, 0.5569612979888916, 0.35210129618644714, -0.7071155309677124, 0.23173196613788605, -0.7578805685043335, -0.5132948756217957, -0.18915653228759766, -2.75423526763916, 0.8000947833061218, -0.5552757382392883, -0.3358103632926941, 0.06898418068885803, 1.0344443321228027, -1.6322928667068481, 1.3849519491195679, 0.006033275742083788, -0.2545913755893707], [0.5027198791503906, -0.6509395241737366, -1.2192153930664062, -0.5629887580871582, 0.8719965219497681, 0.9964970350265503, -0.46862486004829407, 0.09169315546751022, 0.2217913568019867, 0.12519949674606323, 0.7710012197494507, -0.7181510329246521, 0.6261734366416931, 0.23337502777576447, -0.5416505932807922, -0.6715040802955627, -0.7722852230072021, 0.5076755285263062, 1.206318736076355, -1.2789987325668335, 0.2820923924446106, -1.8034480810165405, -0.1798507422208786, -0.3981480598449707, -0.2029813975095749, 0.2908180356025696, -0.04873170703649521, -0.816091775894165, 1.5429809093475342, -0.6566116809844971, 1.0383882522583008, 0.665923535823822, -0.7377293705940247, 0.25447216629981995, -0.40286675095558167, 2.026906967163086, 0.8927187919616699, 0.9706911444664001, 0.5761208534240723, 2.00675892829895, 0.8765117526054382, -0.4537408947944641, -0.6151566505432129, 0.7026431560516357, -0.034525707364082336, 0.8916155695915222, 0.27750658988952637, -0.6837474703788757, -1.383385181427002, 0.20200228691101074, 1.3924635648727417, 1.1074804067611694, -1.0423598289489746, -0.6178589463233948, -1.1307634115219116, -0.8958243131637573, -1.135297417640686, -0.6072507500648499, -0.05444357171654701, 1.9356931447982788, -1.6458566188812256, 1.09184730052948, 0.7202386856079102, 0.29703018069267273, -0.013116699643433094, -0.3840191662311554, 1.081216812133789, 0.13735809922218323, 0.6615166664123535, 0.1342458426952362, -0.12301097810268402, 1.2015479803085327, 0.6859939694404602, 0.10020148009061813, -1.2291585206985474, -0.3119443655014038, 0.5319363474845886, -1.0649504661560059, 0.22075268626213074, -0.541168212890625, -0.20228281617164612, 0.6037038564682007, -1.1240702867507935, -0.8161607384681702, 0.06431113928556442, -1.284877896308899, 0.3649163246154785, 0.35927674174308777, -1.9971626996994019, -1.5170475244522095, 1.5702983140945435, 0.28280267119407654, -0.14455145597457886, -0.9427881240844727, -0.9329249262809753, -0.6420088410377502, 2.0098769664764404, 0.6606324911117554, -0.18922147154808044, 0.31123974919319153, -0.6020106077194214, 0.21263177692890167, -0.5400126576423645, -0.6588801741600037, -0.3713970184326172, -1.131608247756958, -0.08523328602313995, 0.7982116341590881, 0.1828937530517578, 1.518753170967102, 0.7939876317977905, 0.0470159687101841, -0.8757249116897583, -0.8579455614089966, -1.6236406564712524, 1.2427608966827393, 0.535883367061615, -1.40266752243042, -0.7710698246955872, 0.8366270065307617, 1.5081425905227661, -0.9983165264129639, 2.0779340267181396, 0.1611475944519043, 0.75967937707901, -0.3421841263771057, 1.2657443284988403, -0.6018860936164856], [-0.5211887955665588, -0.547295331954956, 0.14667624235153198, -0.4365866482257843, 0.4570433795452118, -0.9329776167869568, 0.18422889709472656, 1.0124033689498901, 1.3415837287902832, -0.4931711256504059, 0.2693236172199249, 0.4206812083721161, -0.3388497531414032, 0.16360224783420563, 0.3696341812610626, -1.6488690376281738, 0.0007076162728480995, -0.29755643010139465, -0.16929659247398376, -0.6889430284500122, -0.022723952308297157, 0.10565928369760513, 0.48073136806488037, -0.7456433176994324, -0.9136908650398254, -0.40159398317337036, 0.34333252906799316, -0.06224533170461655, 0.013721929863095284, 0.47297364473342896, 2.0582799911499023, 0.15320289134979248, 0.553744375705719, -0.7414126396179199, -0.08084999024868011, -1.2294714450836182, -1.0256404876708984, 0.6222898364067078, -0.22613467276096344, -0.5923283100128174, 0.3173319399356842, 1.4702764749526978, 0.4358288049697876, 0.4870620667934418, -0.02507408894598484, -0.93634033203125, -1.956390380859375, 0.9147530198097229, -0.1895913928747177, -1.2494025230407715, 0.44648709893226624, -0.7161734104156494, 1.0796247720718384, -1.0653009414672852, -0.9094762206077576, -0.9986991286277771, 0.5185202360153198, 1.7244762182235718, -0.3752930462360382, -1.3024556636810303, 0.2714911997318268, 0.8961708545684814, -2.3799891471862793, 0.23849007487297058, 0.4437657296657562, -1.1793049573898315, -0.49218228459358215, 1.185272455215454, -0.39670035243034363, 0.086004339158535, -0.4837343394756317, 1.0266691446304321, 0.3729158043861389, -0.7491394877433777, 0.15652215480804443, 0.753256618976593, 1.2101820707321167, -0.480590283870697, -0.6400744318962097, 0.3586310148239136, 1.3978310823440552, 0.7386323809623718, 2.0170583724975586, 0.8985236287117004, -0.2851131558418274, 1.5071375370025635, -1.1428974866867065, -0.2040579915046692, -0.22646668553352356, 0.5482414364814758, -0.27501147985458374, 0.6755599975585938, 0.19843195378780365, 0.8923788666725159, 2.252654552459717, -1.4108773469924927, -1.2101964950561523, -0.9076257944107056, -1.6290459632873535, -0.5105969309806824, 0.66209477186203, 0.14927659928798676, 1.3748425245285034, -1.1211551427841187, 0.5696669816970825, -1.6272602081298828, 0.1469314694404602, 0.3814256191253662, -1.2752474546432495, 1.2313796281814575, -0.7377069592475891, 0.38946402072906494, 1.2358335256576538, 0.9173623323440552, 0.19612881541252136, -1.1617921590805054, -1.5247714519500732, 0.2196142077445984, -0.04277784749865532, 1.9563837051391602, 1.0405904054641724, 0.2661808133125305, -1.0786958932876587, -1.068354606628418, 0.010507858358323574, 0.10018038004636765, 0.9976123571395874, 1.4015018939971924], [0.1775885969400406, 0.06035497784614563, -0.49363669753074646, 0.558007538318634, -0.3830324113368988, 0.7392042875289917, 0.8822240829467773, -0.008306136354804039, 0.1154526025056839, -0.8016557693481445, 1.0005346536636353, 0.3490560054779053, 0.22548946738243103, 0.0716627836227417, -0.6626495122909546, 1.6303144693374634, -1.0365487337112427, 0.4349651038646698, -0.03128618746995926, -0.18558630347251892, 0.3989691436290741, 0.8684450387954712, -0.3008483052253723, 0.9058833122253418, 1.2318856716156006, 0.7434051632881165, 0.8306498527526855, 0.2511017918586731, -0.6556700468063354, 0.08506815135478973, -0.565847635269165, 0.7858911752700806, -0.4571859836578369, -1.0742244720458984, 0.8799030184745789, 0.14723166823387146, -0.9895897507667542, 0.2380468100309372, -1.1834757328033447, 0.3227851390838623, -2.8412418365478516, -0.6443380117416382, 0.07818595319986343, 0.8733209371566772, 0.07988861948251724, -0.6047185063362122, -1.74336576461792, 0.2764880061149597, 1.564343810081482, 0.3028434216976166, -0.15253518521785736, -0.13367284834384918, -0.14653891324996948, 0.3370015025138855, 0.060620080679655075, 0.09934552013874054, -0.4400714337825775, 0.7088242173194885, -0.18797534704208374, 0.3370200991630554, 1.3336166143417358, 0.8218538165092468, 0.9023603796958923, 0.3720594346523285, -2.5030596256256104, 1.2109702825546265, 0.4475099742412567, 0.38983219861984253, -0.038953378796577454, -0.5732563734054565, 0.6608657240867615, -0.9211010336875916, -0.6826369762420654, -0.4644925594329834, 0.6833151578903198, -0.7461205124855042, -1.8674196004867554, 0.5133581757545471, 0.7218248248100281, 0.10420821607112885, 0.5423121452331543, -0.5067028999328613, 0.18528331816196442, 0.1916666030883789, -0.4019263684749603, -0.07610767334699631, -0.35001224279403687, 2.944704055786133, -2.019702911376953, -0.14458675682544708, -1.5762999057769775, 0.8095455169677734, 1.1046050786972046, -0.391532838344574, -0.8689177632331848, -0.16535334289073944, -0.0016820459859445691, -0.8394111394882202, 0.9479902386665344, -0.7766922116279602, 0.7383162975311279, -1.3794715404510498, 0.009026903659105301, 0.4545527398586273, 0.7318992614746094, 1.5028071403503418, 0.017026852816343307, 1.440271258354187, -0.03597687557339668, 0.49814990162849426, 2.1593732833862305, -0.7782505750656128, -0.7423302531242371, -1.1211720705032349, 0.4078686535358429, 1.7526754140853882, -0.35561174154281616, -0.7618862390518188, -0.5252113938331604, -1.1623599529266357, 0.6418766379356384, -0.3198484480381012, 1.7438503503799438, 0.6647515296936035, 0.9554610848426819, -1.0360276699066162, 0.4871828556060791, 1.6984707117080688], [0.27656465768814087, -0.42454078793525696, 0.10496826469898224, 0.5807721018791199, 0.6093945503234863, -0.2937815189361572, -0.07318010926246643, 0.612217903137207, -0.5532875657081604, -0.7345735430717468, -1.2537262439727783, -0.046644777059555054, 1.19756281375885, 1.7280577421188354, 0.3710891306400299, -0.9746482372283936, -1.1741118431091309, -1.31602942943573, -1.111032247543335, 0.20659901201725006, -0.28896263241767883, 0.18538835644721985, 1.2045856714248657, -1.323691964149475, -0.8871764540672302, 0.7022990584373474, -1.2126426696777344, 0.2892399728298187, -0.2140679806470871, -1.292101502418518, 0.10810178518295288, -1.8766754865646362, -0.7148776650428772, -1.800013542175293, 0.8485078811645508, 0.1752810776233673, 0.208948016166687, 0.5804637670516968, 1.3155401945114136, 1.3835885524749756, 0.6385436058044434, 0.31190943717956543, -0.5448883771896362, -1.8660314083099365, 2.2473974227905273, -0.5675438046455383, 2.459538459777832, 1.7768105268478394, 0.0518161803483963, 0.45750004053115845, -0.2155448943376541, -1.1724389791488647, -1.256639838218689, 1.2279342412948608, 0.0712442472577095, -0.5166352391242981, 0.28448712825775146, 1.3561859130859375, 1.099778652191162, -0.9973605871200562, 0.8552778959274292, -0.9749812483787537, -0.8053514361381531, 1.1877285242080688, 0.176249697804451, -0.05286505073308945, 0.014355924911797047, -0.8934417963027954, 0.33981987833976746, 0.14095431566238403, 0.6600956320762634, 0.3016849458217621, -0.7905465364456177, -0.3770507276058197, 1.0408170223236084, 0.433671772480011, 1.0302259922027588, 0.7220792770385742, 1.2592889070510864, -1.6584721803665161, -0.3656245768070221, -0.25148648023605347, -0.43925973773002625, 0.2576056122779846, 1.41917085647583, 0.2981818914413452, 1.6647216081619263, -1.4906948804855347, -0.8951470255851746, -0.5654741525650024, -0.48205655813217163, 1.3092474937438965, -0.023196658119559288, 0.2588590979576111, 1.7019567489624023, 0.6965047717094421, -1.0608553886413574, -0.8749732971191406, 1.2620104551315308, -0.7350913286209106, -0.5108919143676758, 1.199017882347107, -0.5014922022819519, 0.8671984672546387, -0.022912010550498962, 0.33883026242256165, -0.5048550963401794, -1.2022874355316162, -0.5873472690582275, 0.33817294239997864, 0.38079914450645447, -0.36136913299560547, 0.5774487853050232, 1.3718411922454834, -1.6469078063964844, -0.281137615442276, 0.4326449930667877, 0.33710527420043945, 0.45421701669692993, -0.5024898648262024, 0.8461982011795044, 0.9341831207275391, -0.9965648651123047, 1.1184972524642944, -0.100509412586689, 0.3714514672756195, 1.8843318223953247, 0.35221388936042786], [-0.5087497234344482, -0.20816315710544586, 0.9096691608428955, 0.8285753130912781, 0.3339885175228119, 1.1559648513793945, -0.4969002306461334, 0.1321529895067215, 0.1931505799293518, 1.2602967023849487, -1.6004962921142578, -0.36299896240234375, 1.1823281049728394, -1.83961021900177, -0.1347873955965042, 1.1096545457839966, 0.6089102625846863, -0.5256390571594238, -0.24712546169757843, -0.2533797025680542, 0.7236065864562988, 1.3143333196640015, 1.534053921699524, 0.9388130307197571, -0.17132948338985443, 2.825624704360962, -0.44838184118270874, -0.3177163302898407, 0.7243003845214844, -0.7410614490509033, -0.8964625000953674, -1.4778642654418945, 0.5165033936500549, -1.8767634630203247, -0.9911724328994751, 0.8360245823860168, -0.1998121291399002, -0.2613840699195862, -2.0363922119140625, -1.5197798013687134, 0.9421282410621643, -1.8528978824615479, -1.1307703256607056, -0.6682019829750061, -0.015473570674657822, 0.45485594868659973, -0.5320597887039185, -0.19498519599437714, 0.019663797691464424, 0.5892985463142395, 0.12122352421283722, 2.260787010192871, -1.4034053087234497, 0.4297807812690735, 0.17774176597595215, -1.8472243547439575, -0.21034443378448486, 1.9415067434310913, 1.2650434970855713, -0.38263383507728577, 0.37161320447921753, -0.9056031107902527, -1.3706773519515991, -0.16692951321601868, 0.8540920615196228, 0.29488205909729004, 0.350593626499176, -0.09215746074914932, 0.2372819185256958, 1.23361074924469, -0.0679682046175003, -1.1336557865142822, 0.20929628610610962, 0.13759779930114746, 0.7058619856834412, 1.406571388244629, 0.46610280871391296, 0.6180607676506042, -0.10369513928890228, 0.3219771087169647, -0.930413007736206, 0.7301179766654968, -1.1112154722213745, -1.0725083351135254, -1.6672507524490356, 1.0897014141082764, 0.7079061269760132, 0.059233780950307846, 2.6338939666748047, -0.5030233263969421, 1.1608890295028687, -1.427334189414978, 0.6395416855812073, -1.2568082809448242, 1.5462212562561035, -0.6279298067092896, -0.2782725691795349, -0.4383656978607178, -0.8895341157913208, -1.2158111333847046, 0.22949829697608948, -0.2559038996696472, 0.5794938206672668, 0.593126118183136, 0.5525312423706055, -0.950227677822113, 0.029472716152668, -0.8837178945541382, 1.2472147941589355, 0.024228140711784363, 1.7388803958892822, -0.03384416550397873, 0.11553037911653519, -0.6229771375656128, 0.4807387888431549, -1.3218516111373901, -1.0291166305541992, -0.9214391112327576, 0.41787952184677124, -1.6078822612762451, 2.0739998817443848, -0.8569563031196594, -2.0418851375579834, -0.27857115864753723, -0.803764820098877, 0.8648387789726257, -1.2525701522827148, 0.2520713210105896], [-0.9901664853096008, -0.8869702816009521, 0.9000698924064636, -1.419908046722412, -0.3934609293937683, -0.65530925989151, -1.480109453201294, 0.21778903901576996, 2.259565591812134, 1.3325551748275757, 0.09510000795125961, 1.254393219947815, 0.678967297077179, -0.1265895962715149, -1.0350085496902466, 2.0627870559692383, 0.7018755674362183, 0.12190903723239899, -0.7467302083969116, 0.1765744686126709, -0.24881602823734283, 0.36359453201293945, -1.7970205545425415, -1.0114614963531494, -0.5382096767425537, 0.13907930254936218, 0.9051932692527771, -0.46586424112319946, -0.7847146987915039, 1.4777246713638306, 0.43295082449913025, -0.03842279687523842, -0.13826698064804077, -1.9855444431304932, 0.9298115372657776, -2.5174224376678467, -0.36329543590545654, 0.5671262145042419, -0.4272695779800415, 0.6757505536079407, -0.5835705399513245, -1.2738703489303589, 0.8345195651054382, 0.5512457489967346, -1.098583698272705, -0.49199363589286804, 0.8587803840637207, -0.5308149456977844, -0.46189725399017334, 0.9525876045227051, 1.6873869895935059, 1.31345534324646, -1.4323654174804688, -1.1976380348205566, -0.8178059458732605, 0.37728965282440186, -1.1144938468933105, 1.0302221775054932, 0.7065398693084717, -0.9407933354377747, -0.3920694887638092, -0.10148199647665024, 1.188709020614624, 0.9528692364692688, 0.44720083475112915, -0.235800102353096, 0.6393570303916931, -0.8474026918411255, 0.8942315578460693, -0.3007124066352844, 1.6028625965118408, 0.5606048107147217, -0.08261409401893616, 1.583300232887268, -0.31104567646980286, -0.058549802750349045, -0.23323732614517212, 0.694303035736084, -0.06338625401258469, 1.509848952293396, 0.16414032876491547, -0.1572297066450119, 0.9088195562362671, -0.8730819821357727, -0.14235733449459076, -1.9898837804794312, 0.46197739243507385, 0.35423481464385986, 0.02245798334479332, -0.6513861417770386, -0.16407425701618195, -1.8162709474563599, -0.6739518046379089, 2.1187446117401123, 2.37312650680542, -0.27132877707481384, 0.052798107266426086, 1.1707268953323364, 0.7767658829689026, -0.29510101675987244, 0.11411195993423462, -0.4358190894126892, 0.4763675332069397, -0.3509676456451416, 0.4068494737148285, -0.5168138146400452, -2.561037540435791, 0.698843240737915, 0.2967163026332855, -1.8917934894561768, 0.8785662055015564, -0.016172224655747414, -1.3711646795272827, 0.8470868468284607, -2.5581042766571045, 1.6146962642669678, 0.5687377452850342, 0.14485512673854828, 0.23576170206069946, 1.3811744451522827, -1.5851026773452759, -0.6246069669723511, -0.06476616114377975, -0.5912935733795166, 0.10409022867679596, 1.8393927812576294, -0.3164309859275818, 0.5600916743278503], [1.2241404056549072, -1.8912039995193481, 0.45462915301322937, 1.0093178749084473, -1.7310478687286377, -0.4724183678627014, -0.9435540437698364, 1.2080124616622925, 0.36962172389030457, 0.11003462970256805, -0.5256693363189697, 1.4095160961151123, -0.3966018557548523, 1.8690193891525269, 0.6514633297920227, 0.36187416315078735, 1.5022832155227661, 0.11697190254926682, 0.745876669883728, 1.1500166654586792, -0.12574052810668945, 0.7222241759300232, -0.22794407606124878, -0.4677005410194397, -0.5601683855056763, -0.7404592633247375, 0.28669580817222595, -0.5087374448776245, 1.2763748168945312, -0.6274053454399109, -0.7441725730895996, 1.3587560653686523, 0.5073776245117188, -0.22517485916614532, 1.6758557558059692, 0.07278487831354141, 1.1033443212509155, -2.254615068435669, -0.2716893255710602, 0.7774108648300171, 0.09824423491954803, -2.002750873565674, 1.6961878538131714, 0.4816959500312805, -3.0570783615112305, 0.004628901835530996, -0.7958383560180664, 0.10913950204849243, 0.01824866235256195, -0.11255249381065369, 0.09164618700742722, 0.05567308887839317, 0.5439106822013855, 0.38638216257095337, 0.046371132135391235, 1.2182984352111816, 0.08802275359630585, 0.7846323251724243, -2.155360698699951, -0.9481741189956665, -1.7971653938293457, -1.1086161136627197, -0.6705813407897949, 1.7606325149536133, -1.3097585439682007, -1.6988983154296875, -0.41233307123184204, -0.07599753886461258, -1.0856359004974365, 1.071120262145996, 0.6911579370498657, -0.23725664615631104, -0.3007441461086273, -0.531755805015564, 1.5091466903686523, -0.13355636596679688, -0.8905377984046936, -1.5974814891815186, -0.9380766153335571, 0.6218689680099487, 1.9951491355895996, -1.0967484712600708, 1.5103330612182617, -0.3884352445602417, 0.352764368057251, 0.27483096718788147, -0.3623133599758148, -0.17174123227596283, -0.25161781907081604, -0.40221020579338074, 0.5598108172416687, 0.7868035435676575, -0.7661118507385254, 1.055421233177185, -1.2687615156173706, 1.5411994457244873, -1.8437448740005493, 0.6058506369590759, -0.25444814562797546, -0.9464739561080933, -0.3299122154712677, -0.7753753066062927, 1.214371681213379, 0.222465381026268, 0.11471863836050034, 0.14838963747024536, 0.10139085352420807, 0.3329317569732666, 0.10623931139707565, -0.11048226803541183, 0.4858100414276123, 0.776822566986084, -0.6462499499320984, -0.9262245893478394, -0.0028703443240374327, -0.4364677369594574, 0.5215503573417664, -0.7669535279273987, 1.28020179271698, 0.047394759953022, 0.6360417008399963, -0.36210915446281433, 1.5372793674468994, 0.7824316620826721, -0.2135346531867981, 0.8811346888542175, 1.1359162330627441, -0.33610275387763977], [1.3378342390060425, -0.5948139429092407, -0.5915722846984863, -0.2592626214027405, -0.6743910908699036, 0.43278685212135315, -1.359642744064331, -0.1412837952375412, -1.2539488077163696, -1.3224129676818848, 0.20544035732746124, 0.25469449162483215, 0.4510902762413025, 0.7280804514884949, 0.4895568788051605, 1.3908307552337646, 0.1912657618522644, 1.9122538566589355, 0.25211167335510254, 1.0620206594467163, -0.7494754195213318, -1.4218260049819946, 0.9895806312561035, -2.50982928276062, -0.14702782034873962, -2.134840726852417, -1.0843292474746704, 1.8932857513427734, 0.940711259841919, 0.9163946509361267, 0.05839259922504425, 1.5712028741836548, -0.7422766089439392, -0.5218188166618347, -0.8487616777420044, 1.3127681016921997, -0.2276056855916977, 0.47532960772514343, 0.5484076738357544, -1.4007456302642822, -0.8509413599967957, 0.7160811424255371, 0.4690631628036499, -1.7639260292053223, 0.8646480441093445, -0.2819570004940033, 1.0403023958206177, -0.5783858895301819, 0.6698489189147949, 0.727556586265564, -0.5173478126525879, -1.504782795906067, -0.007154344115406275, 1.3917107582092285, 0.5091429352760315, -1.8457242250442505, -0.005377278197556734, -1.7489218711853027, 1.9977704286575317, -0.5302047729492188, -0.5568693280220032, 0.9247210621833801, 3.205634355545044, 1.4479566812515259, -1.764638900756836, -0.017365243285894394, 0.9531481862068176, -0.25519177317619324, -1.023411512374878, -0.0018394084181636572, 0.03420230373740196, -0.386910080909729, 0.33343955874443054, -0.4335557818412781, -1.018092155456543, 2.2339518070220947, 0.3911764919757843, 0.0514385849237442, -1.1688307523727417, -2.011138677597046, 0.21836578845977783, 0.05462266132235527, -0.9603118896484375, -1.4464147090911865, -0.7129002809524536, -0.5499891638755798, -1.7000477313995361, 0.5915897488594055, 1.2983014583587646, 2.091334819793701, -0.29606494307518005, 1.2123034000396729, 0.5378930568695068, -0.18918326497077942, -0.5439935326576233, -0.4431336522102356, -0.4128442704677582, 1.2557207345962524, -0.2408396303653717, -0.13612127304077148, -0.8022617101669312, 0.544037401676178, 0.5610436201095581, -0.9352951049804688, -0.018617814406752586, -0.13497455418109894, 1.4422067403793335, -0.7630881667137146, -0.9534127712249756, 0.04858783632516861, -0.904496431350708, -1.0241777896881104, -0.21917220950126648, -1.5796769857406616, -0.11037302017211914, -0.41879045963287354, 0.5098874568939209, 0.29968345165252686, -1.0936870574951172, 0.5949953198432922, 0.8645495772361755, -1.0078884363174438, -0.021013403311371803, 0.5068768858909607, -1.5602220296859741, -1.347495675086975, -1.1759746074676514, -0.0594957135617733], [0.4336220920085907, 0.9281915426254272, -0.2664167881011963, -0.17262302339076996, -1.3258287906646729, -0.12530776858329773, 0.7684608101844788, 0.753633975982666, 0.5656029582023621, 0.0790737122297287, 0.7428738474845886, 0.07010071724653244, 0.11777576059103012, -0.5483303070068359, -0.26341482996940613, -0.20631077885627747, 1.1889923810958862, -1.4292261600494385, 1.009339690208435, -0.36465510725975037, -1.16751229763031, 0.12911731004714966, -0.41870173811912537, 1.3860357999801636, 1.0378562211990356, -0.009249324910342693, 0.43214723467826843, -0.06707188487052917, 0.29499882459640503, 1.7415379285812378, -0.9659832715988159, 1.5546927452087402, 0.4852816164493561, -2.7219746112823486, 0.6628908514976501, -0.8422412276268005, -0.8315477967262268, -0.7375119924545288, 0.4350717067718506, -0.19511407613754272, -0.12181124836206436, 0.38939929008483887, -0.30113500356674194, 2.0033199787139893, 0.4135552942752838, 0.3857008218765259, 1.1160435676574707, 0.602352499961853, -0.7816261649131775, -0.012463467195630074, 0.10614024102687836, 0.09444674104452133, 1.4841188192367554, -0.31792527437210083, 0.510553777217865, -0.3017442226409912, -0.7529532313346863, 0.944028377532959, 0.4459248185157776, 0.20815342664718628, 0.29300370812416077, -0.23826263844966888, 0.9332671761512756, -2.247735023498535, -0.04787060245871544, -0.5053187012672424, -0.4045732021331787, 1.3473232984542847, 1.0651096105575562, -1.72188401222229, -0.954647958278656, -0.18318963050842285, 0.28849104046821594, 1.026909351348877, -0.5022885799407959, -0.8367801308631897, 1.0957937240600586, -1.0201369524002075, 0.49130329489707947, 0.21616241335868835, -0.7904298901557922, -1.1068322658538818, 1.2339226007461548, -0.4804990589618683, 0.580994188785553, 0.5362223982810974, -1.900059700012207, -0.04822028800845146, 0.20642900466918945, -1.4124553203582764, -0.7289961576461792, -0.654837965965271, 0.8078179955482483, -0.2678716778755188, 0.08225002139806747, -0.9360141158103943, 0.583747923374176, -0.37501809000968933, -0.42895907163619995, 0.1569618284702301, 1.6238254308700562, 0.007973169907927513, 0.4325343668460846, -0.1541176587343216, 0.00199963990598917, -0.8166934847831726, -0.07324357330799103, 0.0016359823057428002, -0.11225605756044388, 1.1111383438110352, -0.854457676410675, -0.30899977684020996, -0.5364142656326294, 0.05311854928731918, -1.719746470451355, -1.2404509782791138, 1.2388489246368408, -0.8542008996009827, -2.1031086444854736, 0.6339455246925354, 0.7002797722816467, -1.8505299091339111, 1.1224768161773682, -1.7151800394058228, -0.3376382291316986, 0.9002172946929932, -0.6181249022483826, 1.7106338739395142], [0.32882729172706604, -0.667702317237854, -0.2209290862083435, -1.1075891256332397, 1.5652259588241577, 0.226410374045372, -0.4143182933330536, -0.2859533429145813, -1.4166022539138794, 0.30886396765708923, -0.4301835894584656, 0.8031443953514099, 1.1475132703781128, -1.834289312362671, 1.7645268440246582, -0.6433315873146057, -0.42341315746307373, 0.8838360905647278, 0.3379875421524048, 1.1270405054092407, 0.5778299570083618, -0.2781842350959778, -0.39207395911216736, 1.1524938344955444, -0.22062058746814728, -0.19275349378585815, -1.5410972833633423, 0.2677605450153351, -0.4670676589012146, -0.8662974238395691, 0.05099334567785263, 0.48555538058280945, -2.179469585418701, -1.5138022899627686, -1.3233639001846313, 0.5258510112762451, -0.23142437636852264, -0.3737109303474426, 0.48176494240760803, 1.8572218418121338, -0.6492233276367188, -1.0905085802078247, -2.383974075317383, -0.6352823376655579, -0.591107964515686, 2.614041328430176, 1.3917968273162842, 2.6060028076171875, -0.5608395934104919, -0.18086330592632294, -0.02025112695991993, 0.856483519077301, 0.5812848210334778, -0.05200723186135292, -1.0195224285125732, 0.013568163849413395, 1.0837664604187012, 0.21926520764827728, 1.268619418144226, -0.9138543605804443, -0.16053104400634766, -0.9747608304023743, -0.18817421793937683, 0.5663440227508545, 0.4580170214176178, 0.11735894531011581, -0.1569480001926422, 0.033350300043821335, 0.05678661912679672, 0.15239977836608887, -0.6562938094139099, 0.43771758675575256, 0.3321959674358368, -0.17627111077308655, -1.2168478965759277, 1.4475113153457642, -1.0537985563278198, -1.4675819873809814, -0.13217154145240784, -0.11725281924009323, 0.40318718552589417, 1.081702470779419, 0.36504316329956055, 0.8504570722579956, 0.9723969101905823, 0.4681374132633209, -0.7277021408081055, 0.4236854314804077, 2.5350260734558105, 1.0426121950149536, 0.7085831761360168, 0.6911405920982361, -2.3435401916503906, 1.7002145051956177, -0.07263597100973129, -0.005948632024228573, -0.8302191495895386, -0.21691206097602844, 0.0011633278336375952, -0.5874432325363159, -0.23721033334732056, -0.36314713954925537, -0.1439327448606491, 2.56801438331604, -1.5241166353225708, -0.6569170951843262, -0.08807235211133957, 0.7264695167541504, 0.23031382262706757, 2.0515544414520264, -0.07244091480970383, -0.7340713143348694, 1.1188242435455322, 0.47233158349990845, -1.3172576427459717, -1.3040999174118042, 0.07283180952072144, 0.39575526118278503, 1.1475989818572998, -1.0279771089553833, -0.6792115569114685, 0.3640610873699188, 0.33529070019721985, 2.205442428588867, 0.05704190954566002, -2.0836310386657715, -0.3049476146697998, -0.6747794151306152], [0.7987890243530273, -1.2366254329681396, 0.9435470700263977, -0.14049537479877472, -0.9767294526100159, 0.24857357144355774, -0.2720690965652466, 0.6720621585845947, -1.0904324054718018, -0.5279877185821533, -1.6516133546829224, -0.30881810188293457, 1.2479852437973022, -1.4370633363723755, 0.051246728748083115, -0.11199092864990234, -1.9328190088272095, 0.6093122959136963, 0.9521611928939819, 1.7119336128234863, -0.3468857705593109, 1.9525060653686523, 0.19912028312683105, 1.0361051559448242, -1.0888721942901611, -0.7226340174674988, -0.4002666771411896, -0.596347451210022, -0.9506891369819641, -0.18219132721424103, -0.3381389081478119, 1.308842420578003, -0.5746435523033142, -0.15270036458969116, -0.4956466257572174, 0.1507035791873932, 0.48895394802093506, -1.5443346500396729, 0.11994273960590363, 0.8259293437004089, -0.3508525788784027, -0.5819219350814819, 1.2442378997802734, 0.42385515570640564, -0.9561456441879272, -0.3007383346557617, -0.13722501695156097, -0.31360822916030884, 1.7263604402542114, -1.6989219188690186, 1.492122769355774, 0.8629629611968994, 0.1428392380475998, 1.4111460447311401, 0.8102352619171143, -0.48123833537101746, -0.5590688586235046, 0.17668646574020386, -1.1808104515075684, 0.9172128438949585, 0.29136574268341064, -0.4470551609992981, -0.7740840911865234, 0.8028327226638794, 0.46032288670539856, 0.7930558323860168, -0.8579411506652832, -2.291823625564575, 0.14565803110599518, -1.3023037910461426, -1.2624377012252808, 0.32755330204963684, 0.3155127763748169, 0.3710591793060303, -0.4032432734966278, -1.6888092756271362, -0.09277143329381943, -0.19900552928447723, -0.8759083151817322, -0.7162295579910278, 0.11471784114837646, -0.471659779548645, -1.9788967370986938, -0.1606580764055252, -0.9815505146980286, -1.4495227336883545, -0.6011474132537842, -0.31312036514282227, -0.665554404258728, 0.13408398628234863, -0.1941346377134323, -1.3494433164596558, -0.6085441708564758, 0.8368613123893738, 1.0984818935394287, 0.2967449426651001, -0.44610127806663513, 2.0720603466033936, 1.124772310256958, -1.023023009300232, -0.2344111204147339, 1.553634524345398, -1.6966298818588257, 0.13261622190475464, -2.7183823585510254, 0.11594255268573761, -1.7742520570755005, 0.8788000345230103, 0.5874079465866089, -0.02105703391134739, -1.5440788269042969, -0.4486396908760071, 1.8580708503723145, -0.595194935798645, 0.8772872090339661, 2.220888614654541, 0.6802460551261902, -0.18302008509635925, 0.24517014622688293, 0.6709724068641663, -0.09853169322013855, -0.8837680220603943, 0.5827907919883728, -0.1880761981010437, 1.0751042366027832, -0.4406180679798126, -1.0740573406219482, -0.22690382599830627], [-0.29758918285369873, -0.24908196926116943, 0.1255388706922531, 1.0285685062408447, -0.7138124704360962, -0.1809324324131012, -0.6634824275970459, 0.3896155059337616, 1.6256132125854492, 1.5060577392578125, -0.22144709527492523, 0.0731920376420021, 1.231227159500122, -0.4713429808616638, -0.5727706551551819, 0.4203380346298218, -0.8290616273880005, 0.2003336399793625, -0.5766350626945496, -0.38420867919921875, -0.13863469660282135, 1.447605848312378, 0.33648326992988586, 0.5023689270019531, -0.31249988079071045, 0.1922333985567093, 0.03938665986061096, 0.7724154591560364, 1.4205228090286255, 1.9357964992523193, 0.20513445138931274, -0.28012681007385254, -0.7294665575027466, 0.03156386315822601, 0.07860836386680603, -0.4140363037586212, -0.27010491490364075, 1.3360390663146973, -0.4569712281227112, -0.28798195719718933, -0.23621536791324615, -0.9422891139984131, 0.2758001685142517, 1.3192075490951538, -0.4111347496509552, -1.7197844982147217, -0.13419297337532043, -0.1721723973751068, -0.047333721071481705, -0.450188010931015, 0.3025374412536621, 1.2682034969329834, -0.5790850520133972, -1.5442291498184204, -0.6166799068450928, 0.14841026067733765, -0.09134821593761444, -1.278326392173767, 0.3812600374221802, 0.29574352502822876, -1.3101222515106201, -2.03802490234375, 0.5042021870613098, 0.835719883441925, -1.4578217267990112, -3.3527369499206543, -0.4282015562057495, -1.0004775524139404, -0.17646220326423645, 0.21198134124279022, 1.0970996618270874, -2.2320058345794678, -0.47377923130989075, 0.7567381262779236, 0.1998952329158783, 0.07551125437021255, 0.33026400208473206, 0.451781690120697, -0.485123872756958, 1.4943077564239502, -1.4414684772491455, 0.256441593170166, 0.001465266803279519, -1.3871550559997559, 1.1892822980880737, -0.4996853768825531, 0.10514821857213974, 0.8644487857818604, -1.429473876953125, 0.14724434912204742, -0.6892127990722656, -1.4226915836334229, 1.323216438293457, -1.6901366710662842, 2.2869865894317627, -0.8712579011917114, 1.3353419303894043, 1.7580716609954834, 0.05030548572540283, -1.03231680393219, -0.5272595882415771, 0.06739204376935959, -0.02167290262877941, 2.1312479972839355, 0.8239810466766357, 0.3635461628437042, 0.5285793542861938, -1.516002893447876, -0.5639108419418335, -1.2300788164138794, 2.117487907409668, 1.140549659729004, 0.4548458456993103, -0.25790560245513916, -0.8744152784347534, 0.8382062911987305, 2.372514009475708, -1.3860340118408203, 0.25147557258605957, 0.6954992413520813, 2.090484380722046, -0.2393483817577362, 0.4957994222640991, -0.5023737549781799, -1.3162544965744019, -1.4619026184082031, -1.0619133710861206, -1.0779552459716797], [1.5074344873428345, 0.9637098908424377, 1.1615523099899292, 0.9841399192810059, 0.7531144618988037, 1.3799242973327637, -0.5776692628860474, 1.2044488191604614, 0.4328003525733948, 0.591914176940918, -0.5680309534072876, -2.2388205528259277, -0.40420717000961304, -0.5870338082313538, -1.1976344585418701, -0.7293894290924072, -0.38823118805885315, 0.44101864099502563, -0.6797179579734802, 0.7308254241943359, 0.18696527183055878, 0.3411244750022888, -1.0608348846435547, 1.8503445386886597, -1.409623146057129, -0.5313829779624939, -0.6462331414222717, -0.2401338368654251, 1.2652653455734253, 0.321087121963501, -0.2871811091899872, -1.3309015035629272, -1.1927629709243774, -1.37529456615448, -1.123026967048645, 0.1783263236284256, 0.047260582447052, 1.9050129652023315, 0.4067777693271637, -2.214811086654663, 2.388899803161621, -1.2117860317230225, 0.16261883080005646, -0.3154792785644531, 0.17861154675483704, 0.14718414843082428, -0.18734285235404968, -2.879868984222412, 0.6642181873321533, 0.6060317158699036, 0.6297414898872375, -0.09720651805400848, 0.30304571986198425, 1.4852652549743652, -1.7153000831604004, 1.1436045169830322, 0.7487451434135437, 1.6506404876708984, 0.6598692536354065, -1.281205654144287, -1.2430609464645386, 0.037152525037527084, 0.862704336643219, 1.4899940490722656, -0.6053754687309265, 0.22252698242664337, -0.04407130554318428, -0.3372665047645569, -0.8839786052703857, 1.0347683429718018, 0.49857062101364136, 0.1671091616153717, -1.735774040222168, -0.020422672852873802, 1.3378897905349731, -1.3262712955474854, 0.7463903427124023, 0.9773009419441223, 0.6985852718353271, -1.2126425504684448, 1.148572325706482, -0.42508649826049805, -0.7894806265830994, 0.7127490043640137, 0.06557086855173111, -0.18666653335094452, 0.21795743703842163, 0.07357543706893921, -0.7075024843215942, -0.6606361269950867, 0.561842143535614, -0.3186090886592865, 0.08627114444971085, 1.2720003128051758, -0.14068855345249176, 0.09518054127693176, 0.024464938789606094, 1.3481459617614746, 0.45813488960266113, -0.3515528738498688, 0.09830482304096222, -1.0204371213912964, 1.3564866781234741, -0.675686776638031, 0.0316695012152195, 0.9488496780395508, 0.25728166103363037, 1.0592230558395386, -0.9675542712211609, 1.4586957693099976, 0.2583216428756714, 0.8921554088592529, -0.1478259563446045, 0.4114735722541809, -1.1503081321716309, 0.13049742579460144, 0.2848053276538849, -1.382531762123108, -0.8389102816581726, 0.8904972672462463, 0.1752949208021164, -0.41388431191444397, -1.403586983680725, -0.09468881785869598, 1.0006532669067383, -0.7614216208457947, -1.9817274808883667, 0.27639949321746826], [-0.5032358765602112, -0.003340542782098055, -0.11965560168027878, 1.3126015663146973, 0.18215055763721466, 0.679131805896759, -0.9096927642822266, 0.8018010854721069, 0.8238006234169006, 0.12344224005937576, -0.1174512729048729, -0.8272635340690613, 0.18648189306259155, -1.5745620727539062, 0.8273571729660034, 0.31797656416893005, -1.5105247497558594, -1.3000644445419312, 0.36106860637664795, 1.2402368783950806, -0.6033441424369812, -0.19400984048843384, 1.3614016771316528, 0.32328277826309204, 0.8847232460975647, 1.1565499305725098, 0.9865489602088928, 0.6984370946884155, 0.1847192496061325, 0.7793717384338379, -1.1327747106552124, 0.6126016974449158, 0.5410255193710327, -1.3088480234146118, 0.663786768913269, 0.03611014783382416, 0.7738773226737976, -0.9060735106468201, -0.7356935739517212, 0.16899770498275757, -1.5676246881484985, 0.7486304044723511, 1.2318861484527588, 0.3706088662147522, 0.3517521023750305, 0.06720516085624695, -0.23969048261642456, -0.5254414677619934, 0.233859583735466, 0.5688246488571167, 0.2030632644891739, -0.843828558921814, -1.6899629831314087, 0.02369229681789875, 0.564449667930603, 0.03876909241080284, 1.4727044105529785, 0.7708796858787537, 0.5390576124191284, 0.8014206886291504, 1.6041570901870728, -0.40729567408561707, 0.6506831049919128, -0.34751248359680176, 1.3731300830841064, -0.886401355266571, -1.0496562719345093, -1.1488077640533447, 0.8654995560646057, -0.27952516078948975, 1.4876741170883179, 0.5089319348335266, -0.2850790023803711, -1.3270350694656372, 0.33518168330192566, 1.1148366928100586, 1.0847196578979492, -0.010544117540121078, 0.7120406031608582, 0.5682191252708435, -0.619132399559021, -0.09077750146389008, 0.46294912695884705, -0.7770837545394897, -0.4377291202545166, 1.6317667961120605, 0.03584270179271698, 1.3287708759307861, 1.2273313999176025, -0.8489993810653687, -1.3675150871276855, 1.3166204690933228, -1.4299019575119019, -1.4104450941085815, 0.7047451734542847, -1.2290605306625366, -0.9694799780845642, 0.7793338298797607, 0.26986023783683777, -0.1532975435256958, -0.1307704895734787, -1.0255608558654785, -0.14339183270931244, -0.2374786138534546, 0.4988630414009094, 0.9278045892715454, 0.494546115398407, -0.7463841438293457, 0.010596255771815777, 0.24651974439620972, 1.1814688444137573, 0.8623521327972412, -0.9940338730812073, 1.2055894136428833, -0.1580866426229477, -1.0765193700790405, -0.012067550793290138, 0.9765331745147705, -2.9146463871002197, -0.8406585454940796, 0.012494250200688839, -0.6719509363174438, 0.9452256560325623, 0.5769603848457336, -0.3149605095386505, 0.2969036400318146, -0.610302746295929, 0.29216840863227844], [1.2855288982391357, 0.002841803478077054, 1.006639838218689, -0.027884354814887047, -0.5094375610351562, -0.11960352957248688, -0.03705023229122162, -0.4643203914165497, 0.5129167437553406, 0.5403375625610352, -0.1542094349861145, 0.4127538502216339, -0.5259762406349182, -0.616432785987854, 0.016894156113266945, -0.7567659616470337, -0.7618122696876526, 0.086773581802845, -0.35171571373939514, 1.1431727409362793, -0.36619192361831665, 0.2190156877040863, -1.5099585056304932, -0.2978112995624542, 1.0493378639221191, -0.7149679660797119, -1.1794606447219849, -1.217300295829773, 0.009880373254418373, 0.7615016102790833, -0.4219297170639038, -1.0627349615097046, -0.12599870562553406, -0.3886890411376953, -1.1930499076843262, -0.3948357403278351, 0.8116588592529297, -0.32413461804389954, 1.3569004535675049, -1.574252963066101, 0.942989706993103, 0.7138884663581848, 0.02634800598025322, -0.11880853772163391, -0.15159671008586884, -1.6668663024902344, 1.6766334772109985, -0.7126299738883972, -0.4260602295398712, -1.467305064201355, 0.35618552565574646, 0.33887460827827454, -0.5290333032608032, -1.9054182767868042, -0.18761491775512695, -0.46853992342948914, 0.2757958173751831, -1.023362398147583, -0.7863461971282959, -1.5999886989593506, -0.018589407205581665, 1.1633599996566772, -0.40457457304000854, 0.9007259607315063, 0.512104868888855, -0.5512697696685791, -0.11986669152975082, -0.7283446192741394, -0.07706040143966675, -2.2403266429901123, 1.263556718826294, -0.47760191559791565, 1.6906813383102417, 1.143147349357605, -0.6216199398040771, -0.0807330310344696, -1.7866793870925903, 0.6086232662200928, 0.9978450536727905, 0.13405737280845642, 1.752837061882019, -0.7816789746284485, 0.7489340901374817, 0.8198813199996948, -0.4639904201030731, 1.0260491371154785, 0.12786531448364258, -0.8687729835510254, 1.3844503164291382, 0.6949285864830017, 0.869166910648346, -0.8319249153137207, 0.08121687173843384, -1.7831534147262573, 0.06924232840538025, -0.2758958637714386, -1.1576308012008667, 0.7776710391044617, -1.972522258758545, 0.12322891503572464, 0.8586305379867554, 0.6843124032020569, -0.15039244294166565, 0.44062793254852295, 0.4549826383590698, -2.9191770553588867, -0.6224925518035889, 1.7535861730575562, -0.3641620874404907, -0.5378440022468567, 0.6748188734054565, 0.00484198983758688, -0.7110225558280945, 0.9061679244041443, -1.5424469709396362, 0.13299603760242462, 0.30674174427986145, -1.1298110485076904, 1.01266610622406, 0.28015202283859253, 0.4651790261268616, 0.2247224897146225, -0.9002721309661865, -0.3949955701828003, -0.25850945711135864, -1.1949869394302368, 0.46336227655410767, 2.087355852127075], [-0.5782905220985413, -0.004968390334397554, 0.8620389699935913, -1.56778883934021, 1.259793996810913, -0.15317197144031525, -0.11753469705581665, -1.0322165489196777, -0.0424027256667614, 0.7999468445777893, -2.0026934146881104, -1.6678804159164429, 1.4145554304122925, -0.5949325561523438, 0.22039711475372314, 0.8751897811889648, -0.8489874005317688, -1.1578532457351685, 0.11523227393627167, -0.7697625160217285, -0.18908870220184326, -0.16064831614494324, 1.908481478691101, 1.6091622114181519, -0.2503412663936615, 0.7748491764068604, -0.13056595623493195, -1.8858458995819092, -0.6795814037322998, 0.813483476638794, -0.5962849855422974, -0.4936445355415344, -0.5996842384338379, -0.6021345853805542, 0.6427406072616577, -0.0923638641834259, 0.2475963681936264, -0.3305060863494873, 1.116005301475525, 0.9809544682502747, 0.9164730906486511, 0.8652864098548889, 1.6293652057647705, 0.9449165463447571, 0.08742530643939972, 0.5311639308929443, -0.9678345322608948, 1.1910113096237183, -0.5599977970123291, -0.569675624370575, 0.10534218698740005, 0.09209167212247849, 1.5749390125274658, -0.4356783330440521, 0.3411317765712738, 1.0174622535705566, 2.32519793510437, -0.5340333580970764, -1.8904272317886353, -2.0087969303131104, 0.17140798270702362, 0.4041251838207245, 1.3266576528549194, 0.11019141972064972, 0.8802706003189087, 1.176878571510315, 0.08808659017086029, 0.651523768901825, 0.5264409780502319, -0.809984564781189, -1.33539879322052, -0.20510199666023254, -0.18148139119148254, -0.26797541975975037, -2.2433252334594727, 1.6520171165466309, -0.06699523329734802, 0.647129476070404, 0.044475264847278595, 0.04005049169063568, 1.4544466733932495, 0.641089677810669, -0.21080338954925537, -0.1081816777586937, -0.5830192565917969, 0.022893907502293587, 0.8449224233627319, -2.1687121391296387, 0.043556395918130875, 0.03035421296954155, -1.6039981842041016, 0.17897705733776093, 0.2578757703304291, 0.0032959168311208487, -0.8858335018157959, 1.050553321838379, 1.2686034440994263, -1.2613893747329712, 1.21591055393219, -0.5165817737579346, 0.7754011154174805, -0.38675156235694885, -1.1419525146484375, -1.0879415273666382, 2.0672149658203125, -1.115775465965271, 0.11193255335092545, -0.6358295679092407, 0.920143187046051, -2.6742045879364014, 0.9014788866043091, 1.34662926197052, 1.6604936122894287, -0.18633145093917847, 1.531242847442627, 0.2297418713569641, 0.8082495927810669, -0.32811233401298523, 0.3815806210041046, -0.023030651733279228, -0.8254783749580383, -0.3740597069263458, -0.07972262054681778, 0.5117880702018738, -0.38648614287376404, -0.14697128534317017, -1.6959433555603027, 1.3176881074905396], [-0.5401821136474609, 0.43391332030296326, 0.9134791493415833, 1.34592604637146, -0.6464133262634277, 1.3680981397628784, 0.5261000394821167, -1.3452180624008179, 0.9716828465461731, 0.7253144383430481, -0.038153816014528275, -1.31004798412323, 0.12763427197933197, 0.7652820944786072, -1.0477758646011353, -0.37539634108543396, -0.8287326693534851, 1.1133776903152466, -0.29609137773513794, 0.07977312803268433, 1.845754623413086, -0.019225554540753365, 1.6698148250579834, 0.4159906506538391, 0.11168563365936279, -0.5794588923454285, -1.3307271003723145, -0.34733590483665466, 0.7145822048187256, -0.0623759962618351, -0.38558775186538696, 1.368929147720337, -0.576217770576477, -0.6884413957595825, 2.4650769233703613, -0.5070674419403076, -1.3075321912765503, 0.5025655031204224, -0.6199051141738892, 0.5260170698165894, 0.041386839002370834, -0.5753629803657532, -1.2985161542892456, 1.1181272268295288, -0.17972561717033386, -0.43484482169151306, 1.8285716772079468, -1.2537990808486938, 1.3244009017944336, -0.2765296697616577, -0.7841223478317261, 0.16040483117103577, -0.9957658052444458, 1.361409068107605, -0.7453945279121399, 0.9718231558799744, -0.5232157111167908, -0.15162020921707153, 0.29211097955703735, -1.508725881576538, -0.08001991361379623, 1.3456370830535889, 1.3025676012039185, -0.3296477794647217, 1.0731914043426514, 0.6248899102210999, 0.4494779109954834, -1.1944142580032349, -0.4461837708950043, 0.5542524456977844, 0.3179912269115448, 0.16951127350330353, 0.1193699836730957, -0.6130433678627014, 0.13494595885276794, -0.5019474029541016, 0.0077836038544774055, -0.12504644691944122, 0.20589132606983185, -0.8693650960922241, 0.8366752862930298, -0.41326239705085754, 0.12116624414920807, 1.6531014442443848, -0.39252060651779175, 0.8892512321472168, -0.14566044509410858, -0.51558518409729, 1.9957199096679688, -0.46510249376296997, 0.41188889741897583, 0.814412534236908, -0.2146858274936676, 1.4031873941421509, -2.3857178688049316, 0.9496837258338928, 0.7029479146003723, 0.7115375399589539, -0.8409765362739563, 0.34346574544906616, -0.3458826243877411, -1.455787181854248, 0.7582194209098816, -1.2821651697158813, 0.015363963320851326, 0.013909822329878807, -1.1492383480072021, -0.8532232642173767, -0.36710718274116516, 0.8199030756950378, 0.6076207160949707, 0.08341756463050842, 0.9296718835830688, 0.4415518045425415, -2.6191251277923584, -1.155652642250061, 0.19979408383369446, 0.15990521013736725, -0.24876363575458527, 0.9710553884506226, -2.2745795249938965, 0.5764601826667786, 0.555464506149292, -0.5121461153030396, 0.9305158853530884, -0.5540280342102051, -0.40703433752059937, 0.8461885452270508], [0.5345526933670044, -1.1290855407714844, 0.4533446431159973, -1.7379117012023926, 0.5390965342521667, 0.8595731258392334, -0.35839641094207764, -0.7231564521789551, 1.820042371749878, 1.3610538244247437, -0.11556629836559296, -0.16306200623512268, 1.0571343898773193, -0.7537211775779724, -0.7193402647972107, -1.5657190084457397, 0.04242367297410965, -1.2884347438812256, 1.5938072204589844, 0.4004996418952942, 1.6133615970611572, -0.5756957530975342, 0.775524377822876, -1.0737656354904175, -0.547045886516571, 1.248000144958496, 1.6123385429382324, 0.7701672315597534, -0.5710331797599792, -0.5253975987434387, -1.1473582983016968, -0.5801324844360352, -1.0933334827423096, 0.510400116443634, 1.986220359802246, -0.007169818971306086, 0.6510542035102844, -0.682456910610199, -0.14631903171539307, -0.9502402544021606, -0.6004624366760254, -1.1658403873443604, -0.10358133167028427, -0.2085476815700531, 0.3680819869041443, -0.8121265172958374, -0.27057430148124695, -0.5779392719268799, -0.7965578436851501, -2.1659250259399414, -0.31210657954216003, 0.31285759806632996, -1.4335825443267822, -0.2498527467250824, -0.7357425093650818, 1.8420445919036865, -0.0016232585767284036, -1.6463509798049927, 0.30928435921669006, 1.3310744762420654, -0.23098976910114288, 1.4588441848754883, 0.930054247379303, 1.0299299955368042, 0.04217320680618286, -0.2561216652393341, -0.9316918849945068, 1.7481136322021484, -1.4088081121444702, -0.545826256275177, 0.5135345458984375, 0.46959611773490906, 0.450784832239151, -0.9169086217880249, -1.6291210651397705, 0.5431144833564758, -1.1829451322555542, -1.103611946105957, 1.4721298217773438, 0.12221391499042511, 1.2517786026000977, -0.8266533017158508, -0.7315818071365356, 0.5297272205352783, 0.8582547903060913, -1.0481164455413818, -1.6356165409088135, 0.1513596475124359, 0.020925795659422874, -0.18246442079544067, -0.7899580597877502, -1.1455998420715332, 2.0530893802642822, -0.4188506007194519, 1.4160083532333374, -1.4521008729934692, -0.1536434292793274, 0.45632338523864746, -1.4270614385604858, -1.4939574003219604, 0.6604012846946716, -0.7497134804725647, -3.0136842727661133, 0.06335270404815674, 0.09688180685043335, -0.12214655429124832, 0.3814624547958374, 0.4065867066383362, -1.7080036401748657, 0.23780126869678497, -0.8438069820404053, -0.6295827031135559, -1.3657681941986084, -0.0043632048182189465, -0.8397501111030579, 0.3008997142314911, -0.6725821495056152, 0.0747278556227684, -0.44838327169418335, 1.4443836212158203, 0.055629223585128784, 0.8320360779762268, -1.5488029718399048, 0.8606275916099548, 1.5083426237106323, 1.2904112339019775, 0.13072772324085236, 1.1580530405044556], [-1.817910075187683, 0.45626962184906006, -0.4247403144836426, 0.24338394403457642, -0.7263693809509277, -0.8628119230270386, 1.8011343479156494, -0.7645900249481201, 1.8726660013198853, -0.3010886013507843, 0.32630786299705505, 2.028127908706665, -0.2773355543613434, 1.110216498374939, 0.4545668661594391, 1.6747914552688599, -0.2982338070869446, -0.20770174264907837, -0.5378998517990112, 0.5022605061531067, 1.359740138053894, 1.375694990158081, 0.225224569439888, -1.4159064292907715, 0.09134388715028763, 0.26405155658721924, -1.6370941400527954, -0.5648118257522583, -0.15448684990406036, -1.29603910446167, 0.02446983940899372, -0.801734209060669, 0.3391760587692261, -0.9832223057746887, 0.06683517247438431, 0.8914731740951538, 1.3349825143814087, -0.3740987777709961, -0.8866808414459229, -2.012333393096924, 0.8641062378883362, -0.7895883917808533, -0.09481481462717056, 0.4534899890422821, -0.4695375859737396, -1.1248294115066528, -2.0377705097198486, 1.4293383359909058, -1.8162819147109985, 0.6818517446517944, -0.23800678551197052, -0.6798643469810486, -0.5511166453361511, 0.32023072242736816, 0.2708474099636078, -0.7690098285675049, -1.097927451133728, -1.996090054512024, -0.33239179849624634, -2.0131328105926514, -1.6669652462005615, 1.710059642791748, -0.11555380374193192, -1.6193861961364746, 1.2329307794570923, -0.5943176746368408, -0.19815035164356232, -0.03099675104022026, -0.3605988025665283, -0.10639891773462296, -0.4976722300052643, -0.24681158363819122, -0.6112176179885864, -0.7256262898445129, -1.6466128826141357, 1.2954704761505127, 1.2093974351882935, 0.8736352324485779, 0.8553639054298401, -0.357724666595459, -0.3551520109176636, 0.9035863876342773, -0.7417911291122437, -0.03382165729999542, -0.3111932575702667, -1.0337399244308472, 0.49196159839630127, -0.3885518014431, -0.20299839973449707, -1.3905786275863647, 0.07513857632875443, -1.3067477941513062, 1.43226957321167, 0.018373413011431694, 0.37151020765304565, -0.17437833547592163, 1.2217477560043335, 2.795158863067627, -1.6076029539108276, -0.8197717070579529, 0.2583700716495514, -0.6355143785476685, 0.2635829746723175, -0.265148401260376, 0.5329124331474304, 1.4266775846481323, 0.07741088420152664, 0.5729529857635498, 0.12629830837249756, -0.36283379793167114, 1.0715279579162598, -0.2443554401397705, -0.7120348215103149, -0.5719737410545349, 0.29483354091644287, 0.7782195210456848, 0.17073313891887665, -1.8443516492843628, 0.34022122621536255, 0.14360526204109192, -0.2064538151025772, -0.05201712250709534, -1.1602991819381714, 0.42513537406921387, 0.4455818831920624, 0.34049326181411743, -1.1672002077102661, -1.426328420639038]], [[-0.008734472095966339, 1.3327014446258545, 0.7636106014251709, -0.72172611951828, -0.01232454739511013, -0.6210401058197021, 0.4611772894859314, -0.6744484305381775, 0.5428645014762878, -0.8509124517440796, 0.12496021389961243, -0.012996514327824116, -1.1891217231750488, -0.0404060035943985, 0.9302752614021301, 0.5238288044929504, 0.9969111084938049, 0.8374224305152893, 0.41341885924339294, 0.6195178627967834, -0.6441766023635864, -2.1219849586486816, 0.79168701171875, -0.23699529469013214, 1.0242860317230225, -1.5693995952606201, -0.9211670160293579, -0.35362452268600464, -0.5392382144927979, 0.5879894495010376, -0.1407274305820465, 0.396860271692276, 0.5416373610496521, -0.04508912190794945, -1.7869246006011963, 0.7160383462905884, 0.24941830337047577, -0.39106252789497375, 0.2943130135536194, -1.1436800956726074, 1.250655174255371, -0.36027494072914124, -0.9643168449401855, 1.4667890071868896, -0.3843594193458557, -1.1657770872116089, 0.013195632956922054, 1.8061721324920654, 0.07621876150369644, -0.4879232943058014, 2.572579860687256, -0.5146059989929199, -0.5017732381820679, -0.3616540729999542, -0.11913440376520157, 2.1090292930603027, 0.5436949729919434, -1.3400276899337769, 0.5644622445106506, 1.9265875816345215, -0.6006386876106262, -0.36005014181137085, 0.36553630232810974, 1.0787760019302368, 1.5031284093856812, 0.06266529858112335, 1.4383362531661987, 1.2298377752304077, 1.4836969375610352, 1.4211469888687134, -0.06527309119701385, -0.3625384569168091, 0.17391802370548248, 0.16333161294460297, -0.8734010457992554, 1.799680233001709, -0.34696751832962036, -0.6914756894111633, 0.7288358807563782, -0.3485962748527527, 0.15229497849941254, 1.1261993646621704, -1.398245096206665, -0.6838545203208923, -0.6500096917152405, -0.9244275093078613, 0.3359050750732422, 0.41457316279411316, 0.840599536895752, 0.8072686791419983, 0.5023140907287598, -0.44468459486961365, -0.8884683847427368, -0.38477733731269836, -1.0431853532791138, -0.8313685059547424, -0.08601560443639755, 0.12885038554668427, -0.5208423137664795, -1.678462028503418, -0.07959253340959549, 1.6202950477600098, 0.36210083961486816, 0.08306003361940384, 0.28839337825775146, -0.756573498249054, -0.634467601776123, -1.6017777919769287, 0.9289376735687256, 0.9688115119934082, 0.4077104330062866, 0.8200308084487915, 0.09414219111204147, 1.2285252809524536, -0.3973698616027832, 0.10827232152223587, 0.013519617728888988, 1.7625746726989746, 0.8261007070541382, 0.7560480237007141, 0.15873074531555176, -0.3243078589439392, 0.46194717288017273, 1.15763521194458, -1.4298715591430664, -0.015359125100076199, -1.3598018884658813, 0.07375548034906387], [0.2606581747531891, 0.8414242267608643, 0.31666263937950134, 0.22333458065986633, 0.03233388438820839, -1.0368787050247192, 0.4019075036048889, 1.8432857990264893, -0.542927622795105, 0.6830213665962219, 0.8489882349967957, -1.5586882829666138, -0.6020258069038391, 1.1416980028152466, -0.08318334072828293, -0.372561514377594, 0.1702144593000412, 0.39788132905960083, -0.08217647671699524, -0.85304856300354, 0.206252783536911, 0.3951651155948639, -2.183959484100342, 0.2151390165090561, -0.42467641830444336, -1.4834073781967163, 1.1240121126174927, 0.5945367217063904, -0.6675605177879333, 1.4920765161514282, -0.888914942741394, -1.4764511585235596, 0.9229094386100769, 1.0581650733947754, 0.4091332256793976, 0.6479783654212952, -0.054461654275655746, -0.7145329713821411, 0.9622433185577393, -1.1812925338745117, -1.7381466627120972, 0.35949012637138367, -0.7683721780776978, 0.8175809383392334, -0.1694502830505371, 0.31303101778030396, -0.6494219303131104, -0.72621089220047, -0.6517340540885925, -1.0282940864562988, 0.5598081946372986, -1.3910211324691772, -0.11608559638261795, -0.2760070860385895, -0.6692334413528442, -1.0017914772033691, 1.8710525035858154, -0.8767194747924805, 1.392391562461853, -0.2013889104127884, 1.2614818811416626, 0.06734522432088852, -1.4770607948303223, 1.2479937076568604, 2.136291265487671, -0.9020664691925049, 1.415140986442566, 0.25434720516204834, 0.5379663705825806, -0.10474051535129547, -0.3761751353740692, -0.7002751231193542, 0.14271336793899536, 0.8437663316726685, -1.13504159450531, -0.5161663889884949, -0.4238704741001129, 0.474547415971756, -1.4555808305740356, -1.2436357736587524, -0.6759857535362244, 0.4588454067707062, -0.1518252044916153, -0.4774424135684967, -0.5730368494987488, -0.46677207946777344, -0.5237608551979065, -0.2197660207748413, 2.564821243286133, -2.4918015003204346, 0.46941009163856506, 1.4637387990951538, -1.024289846420288, -0.004826866555958986, -0.5434427261352539, 0.024378882721066475, 0.40022048354148865, 0.6595008969306946, 1.2075531482696533, -0.8533998131752014, 1.5706217288970947, -0.09712318331003189, -0.6230878233909607, -1.7726902961730957, -0.8817793130874634, 0.7378814220428467, -1.3669406175613403, -2.662369966506958, -0.8578222990036011, 0.036790914833545685, 0.3000647723674774, 0.11723192036151886, 0.7522153258323669, 1.3489652872085571, -0.41054847836494446, 0.08010886609554291, 0.3640812337398529, 0.9714763760566711, -0.7710586786270142, 2.1967554092407227, 0.6044061183929443, 0.2784558832645416, 1.0431804656982422, -0.0031165415421128273, 0.5231770277023315, 0.5278766751289368, -1.437320351600647, 0.009078197181224823], [-0.6898974180221558, -0.2599237561225891, 0.5377057194709778, -0.010250476188957691, -0.5610891580581665, -0.6278742551803589, -0.584911048412323, -1.3396592140197754, 0.6441413164138794, 0.5933056473731995, -1.0217877626419067, -0.41887471079826355, 0.7220399975776672, -0.1029280424118042, -1.4406052827835083, 0.4923602044582367, 1.3452962636947632, 0.029089441522955894, 0.7483141422271729, 0.7961235642433167, 1.1452525854110718, 0.7701051831245422, 0.1777380108833313, 0.2737811505794525, -1.1273070573806763, -0.6768650412559509, -1.0738701820373535, -0.621552586555481, 0.4800385534763336, 1.4280345439910889, 0.8315539956092834, 0.49162259697914124, -1.5914454460144043, 0.4764477014541626, -0.2437644898891449, -0.5872409343719482, 2.4238297939300537, 0.3488822281360626, 0.387087345123291, 1.207189679145813, -1.2483075857162476, -0.3326285183429718, -1.1753205060958862, -1.7177330255508423, 0.2059338390827179, 1.4261442422866821, -0.13752123713493347, -0.9183082580566406, -1.4434038400650024, 1.1495513916015625, 1.616360068321228, 1.8502479791641235, 0.34183669090270996, 0.2997530698776245, 0.4102901518344879, -1.3436199426651, 0.609433650970459, 0.011637352406978607, -0.0068673305213451385, -1.1072099208831787, -0.30893489718437195, 0.09499496221542358, -0.8691544532775879, -0.5614049434661865, -1.834309697151184, 1.0262935161590576, -0.9941180944442749, -1.2103230953216553, 0.47254547476768494, -0.07261226326227188, 0.09126096218824387, -0.8749695420265198, 1.3224117755889893, 0.7153043150901794, -2.2000834941864014, 0.21900460124015808, -0.30728912353515625, -0.8359090685844421, 1.365438461303711, -0.13858072459697723, 0.13151997327804565, 0.7573559880256653, 0.10766952484846115, -0.2155742645263672, -0.905927836894989, -1.0484561920166016, 1.152300238609314, -0.5493123531341553, -0.32671934366226196, 0.41379043459892273, 0.2472054809331894, 1.0524009466171265, 0.9130436182022095, 1.0047814846038818, 1.0344452857971191, -1.244024634361267, 3.218618631362915, 0.7518867254257202, -0.565444827079773, 0.3282626271247864, -0.8119485974311829, -1.0366325378417969, -0.9479910135269165, 0.5570701360702515, 0.3767800033092499, -0.7614184021949768, 0.3717263340950012, 0.7243232131004333, -0.7273723483085632, 0.9470093250274658, 0.9505984783172607, 0.35409122705459595, -1.375520944595337, 0.6176354885101318, -0.11711500585079193, -1.9632917642593384, -0.4194100797176361, -0.1421566903591156, -0.34577786922454834, -0.8143128752708435, 0.12127165496349335, 0.9019173383712769, 1.4508556127548218, -0.1612803339958191, -1.1534743309020996, 0.22287674248218536, 1.3329106569290161, -1.6668914556503296], [1.8934754133224487, -0.19645310938358307, -0.1105315163731575, -0.47450897097587585, 1.8490791320800781, 0.9845614433288574, 1.6644858121871948, -0.36195963621139526, -0.7469369769096375, -0.8836742043495178, 0.47436943650245667, 0.49780771136283875, -1.4801254272460938, 2.4291646480560303, -1.1763427257537842, 0.8570027351379395, -0.8436379432678223, 0.44965696334838867, -0.09291200339794159, -1.0657973289489746, 0.5389553308486938, -0.23796845972537994, 1.2700080871582031, -0.34392663836479187, 0.12366127967834473, 1.4477403163909912, 1.8567208051681519, -0.8355449438095093, -0.4957101345062256, 0.4090616703033447, 1.3860769271850586, -0.7237699031829834, 0.8355410099029541, 0.5179035067558289, -2.2798774242401123, 0.7828624248504639, -0.28422611951828003, 0.3582786023616791, -1.4971730709075928, -0.44801196455955505, 1.310096025466919, -0.09028022736310959, -0.785057008266449, 0.13536258041858673, -0.9385042786598206, -1.4055012464523315, -0.3104725778102875, -0.2095673680305481, -1.313050389289856, 1.7527153491973877, -1.1598519086837769, -1.0087610483169556, 0.06418208032846451, -0.19440577924251556, -0.5296162366867065, -0.39432841539382935, 0.6538923382759094, -1.5044108629226685, 0.24526767432689667, 0.7487166523933411, -1.255448818206787, 0.0741446241736412, 0.17615976929664612, -1.4775855541229248, -0.08563821762800217, 0.7493809461593628, -0.012797525152564049, 0.26419946551322937, 0.7007752656936646, 0.1338832825422287, -0.08343549072742462, 0.40686434507369995, -1.5178934335708618, -1.6657721996307373, -0.9310683012008667, 1.134996771812439, 1.453808069229126, 0.7717480063438416, 1.8562415838241577, 1.0018723011016846, -0.21897824108600616, -0.9656882882118225, 0.22093448042869568, 0.22399598360061646, -1.5342803001403809, 0.055404722690582275, -0.18678328394889832, -0.3500637114048004, -1.1303414106369019, 0.5768859386444092, 1.1774979829788208, 0.15401263535022736, 1.3574388027191162, 1.6910598278045654, -2.5511693954467773, 0.4203789532184601, -1.3585631847381592, 3.0337061882019043, 0.5830872058868408, 1.0782301425933838, 0.5061330795288086, 1.3003133535385132, 1.0630208253860474, -0.9454964995384216, -1.0911909341812134, 0.6852348446846008, 2.743861436843872, -0.13070088624954224, -0.5753509998321533, -0.11062157899141312, -0.011471397243440151, -0.020468369126319885, 0.5566033720970154, -2.4886276721954346, 0.711162269115448, -0.46018025279045105, -0.13873428106307983, 0.7443571090698242, -0.396048367023468, -1.639426350593567, -0.5586645603179932, -2.242363691329956, -0.9157297015190125, -0.9958391189575195, 1.349919080734253, -0.33713120222091675, 0.12626758217811584, -2.2725985050201416], [-1.1531015634536743, -0.6416305303573608, 0.9057729840278625, -2.9811198711395264, 0.4788093566894531, 0.045247334986925125, 0.4282758831977844, 0.6111586689949036, -1.9385473728179932, 0.39631426334381104, 0.08752866089344025, -0.848315954208374, -1.0454685688018799, 2.185856342315674, 0.13146968185901642, 1.4433071613311768, 0.9716131091117859, -0.07205548882484436, -1.087441325187683, 0.41629523038864136, 0.6284359693527222, 0.18074657022953033, 0.5464305281639099, 0.23018072545528412, -1.4632973670959473, -1.5321730375289917, 0.9843500256538391, 2.595773220062256, -0.42482128739356995, -0.4999997019767761, 0.7538524270057678, -0.7813687324523926, 0.6431002020835876, -0.6311472058296204, 0.39643365144729614, 1.1835304498672485, -1.1046446561813354, 0.3467981517314911, -0.20774728059768677, -0.797901451587677, -0.6397404670715332, -0.9832401275634766, 1.2595936059951782, -0.44123584032058716, 0.2477577030658722, 0.528614342212677, -1.951812505722046, 1.1215656995773315, -0.09492133557796478, -0.8255552053451538, 0.8651641607284546, 2.3603551387786865, 0.607792854309082, 0.30405840277671814, 0.8066943287849426, 2.0513689517974854, -0.619776725769043, 0.7431524991989136, 0.8228927850723267, -0.8235979676246643, 0.3808475136756897, -0.2750759422779083, -1.1560174226760864, -0.6178146600723267, 2.2000269889831543, 0.6905520558357239, -0.8830323815345764, 1.5553183555603027, 0.7509939074516296, -0.2537422478199005, -1.0418405532836914, -0.14702825248241425, -1.168997883796692, -0.6316556930541992, -0.4142380356788635, -0.5514976382255554, -1.8876147270202637, 1.469718337059021, 0.18531787395477295, 1.0779504776000977, -0.025439605116844177, 1.6553795337677002, 0.24920515716075897, -0.5723311901092529, -0.07633151113986969, -1.0264819860458374, 0.020365530624985695, -1.098708987236023, -0.8836859464645386, 0.9814292192459106, -0.6911339163780212, 0.7833153605461121, 1.1471375226974487, -0.6347141265869141, -1.1105259656906128, -0.9515364170074463, 0.22681067883968353, -0.18686999380588531, 1.2726881504058838, -1.277477741241455, -0.7554068565368652, -0.720978319644928, -0.1795884668827057, -0.42691007256507874, -1.4875432252883911, -2.0033345222473145, -0.02502250112593174, -1.1425690650939941, 1.7443668842315674, -0.2006857693195343, -0.8698492646217346, -0.08139725029468536, -0.08245261758565903, 0.09630041569471359, 0.32548245787620544, -1.7752301692962646, -0.7541589140892029, 0.06577272713184357, 2.1439170837402344, -0.805996298789978, -0.4788650572299957, 1.6379059553146362, -0.01814660243690014, -1.3118150234222412, 0.8850181698799133, -0.40492939949035645, -1.6085008382797241, -0.6954967379570007], [1.29238760471344, 1.777314305305481, -1.2966244220733643, -0.4891332983970642, 0.0570342056453228, 0.8210768699645996, -1.236331820487976, 0.9739052653312683, 0.3963155746459961, -0.3817862868309021, 0.7046349048614502, -0.4863349497318268, 0.7615129947662354, -1.0198800563812256, -1.0095120668411255, 0.7639187574386597, -0.5635312795639038, -0.0766671746969223, 1.3580011129379272, 1.699464201927185, 0.8697525262832642, -1.092437982559204, 0.17793908715248108, -0.5995723605155945, 0.7827816605567932, -0.6719598174095154, -0.16640537977218628, -0.30549541115760803, -1.0391414165496826, 2.529097557067871, -0.16084304451942444, -0.9141662120819092, 1.6339390277862549, -0.5515540242195129, -0.30249354243278503, 0.10758636146783829, 0.6284343600273132, 0.6889804005622864, 1.0829449892044067, 0.0604410283267498, -0.6148643493652344, -0.3139069378376007, -1.861104130744934, -0.05090972036123276, 0.6800671219825745, 1.825537919998169, 0.2420239895582199, -0.24265915155410767, 0.07845542579889297, -1.693101167678833, 0.4854469299316406, 0.4595872163772583, -0.08858601748943329, 1.1428639888763428, -1.2989174127578735, 0.3874621093273163, 0.2203986644744873, 0.1803501695394516, 0.08561106026172638, -0.7690732479095459, -0.4177524745464325, -0.19636784493923187, -1.4195038080215454, -0.9197543263435364, 0.12536950409412384, -1.0767014026641846, -1.3118693828582764, -0.660323441028595, 1.7444087266921997, 0.4762945771217346, 0.7949167490005493, -1.3207507133483887, -0.706831693649292, -0.8857674598693848, 1.0975443124771118, -0.7315890789031982, -0.3885708749294281, 0.12340714037418365, 1.4850865602493286, -0.005924754776060581, 0.9259847402572632, -0.2533111274242401, 1.3256675004959106, 0.3604159951210022, -0.7645414471626282, 0.3017559051513672, -0.18914054334163666, -0.36451053619384766, 0.6581145524978638, -0.8695579767227173, -0.2584514617919922, -0.6871788501739502, -1.9555736780166626, -0.9132155179977417, 0.9897205233573914, 0.3825764060020447, -0.362735778093338, 0.7770390510559082, -1.9605461359024048, 0.09192486107349396, -0.258851021528244, 2.992511510848999, 0.8309739828109741, -0.8063534498214722, 1.762874960899353, -1.6106387376785278, -0.12426958233118057, -1.1930456161499023, -0.3290063440799713, -0.33241719007492065, 0.3194139301776886, -2.1704399585723877, 1.0817724466323853, 1.0220122337341309, -0.053571559488773346, 0.7758675813674927, 0.2110559195280075, -0.7628532648086548, 0.799382209777832, 1.4612061977386475, -0.8639193773269653, 0.33946213126182556, -1.3278411626815796, 2.192591905593872, 0.11590206623077393, -2.333645820617676, -0.9907547235488892, 0.6430383920669556], [0.45817631483078003, -1.1343663930892944, 2.960028648376465, -1.8767400979995728, 0.6792322993278503, 0.9102358222007751, 0.12089747935533524, -0.5245459079742432, -2.2927401065826416, -2.876542091369629, -1.29815673828125, -0.009412211365997791, 0.43798011541366577, 0.13663052022457123, -0.8422201871871948, 0.1330462396144867, -0.16000711917877197, 0.3624366521835327, -0.07077796757221222, 0.22027982771396637, 2.4619946479797363, -1.3521441221237183, 1.173092007637024, 1.5090559720993042, -1.1880443096160889, 0.45933231711387634, -1.11392343044281, -0.767874002456665, -0.4195460379123688, -0.5437936186790466, -1.5875599384307861, -1.7043697834014893, 0.46937865018844604, -0.07219071686267853, -0.6711511611938477, -1.0394505262374878, 0.30828288197517395, -0.3107098639011383, 0.37157779932022095, -0.5188207626342773, 1.6104680299758911, 0.10626578330993652, -1.1349692344665527, -0.2828681170940399, -0.8766407370567322, -2.1193289756774902, -1.4174782037734985, 1.3759372234344482, -1.1408392190933228, -0.5711217522621155, 0.888237476348877, -0.5518305897712708, -0.63517826795578, -0.7466048002243042, -0.35272929072380066, 0.3530017137527466, 0.7049298882484436, -0.531787097454071, 0.13467249274253845, -0.5398229360580444, 0.7650309205055237, -0.1313951313495636, -1.568372130393982, 0.013945513404905796, 0.9006273746490479, 1.4475172758102417, -1.814875602722168, 1.148776650428772, 0.5554717779159546, -0.23640082776546478, -2.2869160175323486, -0.5524595975875854, 0.7217609882354736, 0.9551239609718323, -3.067800998687744, 0.1027887612581253, -1.2490838766098022, -0.15880827605724335, -0.5617021322250366, 1.2830721139907837, 0.4872911274433136, 0.31820619106292725, -1.6111043691635132, 0.29469698667526245, -0.17593452334403992, 0.3431309163570404, 0.7248058319091797, 0.878054141998291, 1.5793256759643555, -2.084932327270508, -0.3128654658794403, -0.7157660126686096, -1.6433517932891846, 1.0148664712905884, 1.2619966268539429, 0.4228024184703827, 0.3917112946510315, -0.5582108497619629, -0.5362679958343506, -0.6421589255332947, 1.297332525253296, 0.4166681170463562, 1.072312355041504, 1.4767783880233765, 0.9297153949737549, -0.8665104508399963, -0.5698213577270508, 0.6056374907493591, -0.4905220866203308, 1.066330075263977, -1.3788775205612183, 1.2267414331436157, 2.509199619293213, -0.13151152431964874, 0.4661317467689514, -0.08257406949996948, 0.22806310653686523, -0.3267832100391388, -2.0921356678009033, -0.3047087490558624, -0.17373129725456238, -0.22596034407615662, 0.339225172996521, -0.25196948647499084, 0.24693602323532104, -1.6226608753204346, -1.9675021171569824, -0.22794915735721588], [0.38196393847465515, -4.0293169021606445, 0.2165779322385788, 1.1731115579605103, -0.46504515409469604, -0.7643314003944397, -0.9301482439041138, 0.18508148193359375, -0.49202144145965576, 0.1427929699420929, 0.06505756825208664, -0.08312363177537918, -1.378511667251587, -0.8982268571853638, 0.2652210295200348, -1.3968596458435059, 0.4880077540874481, -0.14110340178012848, 1.6271212100982666, 0.3554014265537262, 0.19520540535449982, -0.21773652732372284, 0.10559049993753433, 1.2218247652053833, 0.344488263130188, 0.9308543801307678, 0.3661198019981384, 0.8991259336471558, 0.10556083917617798, 1.9763129949569702, -0.21207694709300995, 1.0659795999526978, -0.18058764934539795, 0.6737472414970398, 0.013778620399534702, -0.326911985874176, -1.100640058517456, 0.6180883049964905, -1.3894424438476562, 0.5114205479621887, -2.007420778274536, -1.222563624382019, -0.19789248704910278, 0.9931193590164185, 1.0184446573257446, 0.9597187042236328, -0.03793143853545189, 0.6720367670059204, 0.36705437302589417, 0.3544791340827942, 0.6099762320518494, -0.9838517904281616, 1.095320224761963, 1.3782730102539062, 1.9833307266235352, 0.9663995504379272, -1.8877722024917603, 2.3298771381378174, -0.7987380027770996, -0.3407191038131714, 0.07080218195915222, -1.212708592414856, 0.8019533753395081, -0.6701036691665649, -0.29844915866851807, -0.4607078433036804, -0.09307020157575607, -1.2140899896621704, -0.9746103286743164, 0.20212915539741516, 0.34054067730903625, 0.1484445333480835, 0.7681865096092224, -1.715949535369873, -2.7413737773895264, 1.3667826652526855, -0.6727025508880615, 0.01563955470919609, -0.47791817784309387, -0.11290350556373596, -1.1080825328826904, 0.9761542677879333, -1.2900365591049194, 1.8130892515182495, 0.35199856758117676, -1.2989591360092163, 0.061350900679826736, 1.478509545326233, -0.2329508662223816, -0.3879130184650421, -1.3972246646881104, 1.5094759464263916, 1.3917204141616821, 0.7293119430541992, -1.1130625009536743, 0.6683861017227173, 0.8372161388397217, -1.2897440195083618, -0.568244457244873, 0.6749973893165588, 0.2594878375530243, 1.3675248622894287, 0.8103231191635132, 1.0721163749694824, -0.4157263934612274, 0.3674667477607727, 1.3488417863845825, -0.8042240142822266, -0.5164837837219238, 0.5034018754959106, -0.5859534740447998, 0.14122281968593597, -0.45749199390411377, 0.4730214774608612, -0.012264042161405087, -1.4429658651351929, -0.8108083009719849, 1.0198520421981812, -1.4093073606491089, -0.04572487995028496, -1.0388340950012207, -0.49959632754325867, 0.6824061274528503, -0.5278589129447937, -1.1812113523483276, 1.025334358215332, -0.05852193012833595, -0.12495677918195724], [0.21397580206394196, 1.04300057888031, 0.01771712489426136, -0.5766514539718628, -0.9962456226348877, -0.2579846680164337, -2.206463575363159, 1.881101369857788, 0.8155649304389954, 0.11217507719993591, -0.5676318407058716, -0.024248402565717697, -0.4997478723526001, -0.5487316846847534, 0.12937340140342712, 0.525463342666626, -0.42658731341362, -0.17685404419898987, -1.8440309762954712, 0.6206604838371277, -0.7879708409309387, 0.9219846129417419, 0.342409610748291, 0.6172705888748169, 0.5789461135864258, 1.4528980255126953, 0.15073134005069733, -0.19297648966312408, 0.08952798694372177, 0.10053087770938873, 2.115978479385376, 0.8907857537269592, -0.40778911113739014, -1.7003086805343628, -0.045513126999139786, 2.5463790893554688, -1.7290165424346924, 0.6639522314071655, -0.5266297459602356, 0.700416088104248, -0.5085262656211853, -0.5572603344917297, 1.4666918516159058, 1.3126089572906494, 0.47848591208457947, 0.008353953249752522, -0.06342098861932755, -0.28103139996528625, -1.3038564920425415, 1.713202714920044, -1.1665886640548706, -0.640962541103363, 0.19832904636859894, -1.6580510139465332, -0.943386435508728, -1.4146281480789185, -0.07088048756122589, -0.976543128490448, -0.673825204372406, -0.04200133681297302, -0.7657538652420044, -0.818664014339447, -0.11122985929250717, -1.5949535369873047, -0.42783546447753906, -1.2740715742111206, 1.794919490814209, -0.7020302414894104, -1.4909031391143799, 0.1392456740140915, -0.4503861665725708, -0.6637601256370544, -0.4395816922187805, -0.6690291166305542, -0.05549001693725586, -0.03447756916284561, 1.6388204097747803, -2.667541265487671, -0.444854736328125, -0.1643296778202057, 0.4471839666366577, -0.9757073521614075, -0.7291184067726135, 0.258943647146225, 0.40036752820014954, -0.8882929086685181, -1.4492368698120117, -0.9935073852539062, -0.4977549612522125, 0.7120065689086914, -0.4077230393886566, -1.0935362577438354, -1.0185301303863525, -0.31578904390335083, -1.7344989776611328, -0.18519684672355652, -1.0149096250534058, -0.5223851203918457, 0.004391548223793507, 0.8719283938407898, 1.0348347425460815, 0.20328988134860992, -0.012500983662903309, -1.150874137878418, 0.8818439841270447, -0.5024991035461426, -0.8288289308547974, -1.5975147485733032, 0.6913832426071167, -0.012783615849912167, 0.4266323745250702, 0.12819251418113708, -1.4398324489593506, -0.27568626403808594, 0.6697953343391418, -0.5132311582565308, 0.8131757974624634, -0.10684414207935333, -0.2331843078136444, 0.8957593441009521, -0.9669171571731567, 0.15613612532615662, -0.7138321995735168, 0.41778257489204407, -0.9385973215103149, 2.0093696117401123, 1.9808623790740967, -0.29653599858283997], [-0.08182678371667862, -0.48493948578834534, 0.18763086199760437, 0.9294368028640747, -0.9128953814506531, -1.5149942636489868, -0.40488582849502563, 1.139797329902649, -1.461622714996338, 0.6385725736618042, 0.9054732918739319, 1.937037706375122, 0.7854561805725098, -1.0751487016677856, 0.8586612343788147, 0.42941802740097046, 0.46127331256866455, -1.4242466688156128, 0.05260614678263664, -0.07367103546857834, 1.118442416191101, -0.07739292085170746, 0.1571149230003357, 0.45338910818099976, -0.9029029011726379, 1.2067281007766724, 1.3478806018829346, 0.9218006730079651, 1.3926069736480713, 0.5389318466186523, -0.2376932054758072, -0.27339306473731995, -1.5061475038528442, -0.7459867596626282, -0.7702087759971619, 0.9882944226264954, -2.4472641944885254, -0.6490247845649719, -0.9626086950302124, 1.3274579048156738, 0.2501228153705597, 0.348891943693161, 1.1835092306137085, -0.18359456956386566, 0.56098872423172, 0.08150264620780945, -0.4457549750804901, 0.2872127294540405, 0.1296558976173401, -0.29945966601371765, -1.0931063890457153, 0.12992259860038757, 1.1483492851257324, 0.31132155656814575, 0.2510010600090027, 0.39023661613464355, -0.10019201785326004, 0.03291346877813339, -0.17897717654705048, -0.9532710313796997, 0.30656734108924866, -0.4180547297000885, -0.49475717544555664, 1.9833308458328247, 1.5232772827148438, -0.7392292618751526, -0.32534530758857727, -0.411975234746933, 1.260818600654602, -1.0562106370925903, 0.955786943435669, 0.17055027186870575, 0.03448905795812607, -0.2367940992116928, -1.366490125656128, 0.7627788782119751, -0.17449909448623657, -0.23430530726909637, 0.3459465205669403, 0.236142098903656, 1.6138371229171753, 0.43457332253456116, 0.153771311044693, -0.6715971231460571, 1.5619525909423828, 1.785581111907959, -0.4614187479019165, -0.5586066246032715, -0.2603716254234314, 1.8634088039398193, -0.4094236493110657, 0.14373625814914703, 0.44159942865371704, -0.6589901447296143, -0.5816476345062256, -0.3868075907230377, 0.6791015267372131, -2.1948022842407227, -0.08772704750299454, -2.005201578140259, 0.23708394169807434, 1.4337480068206787, -1.2070773839950562, 0.20394378900527954, -1.6032013893127441, -0.06501996517181396, -0.08954913169145584, 1.0012315511703491, 0.5678297877311707, 0.651312530040741, -0.7464489340782166, -0.24807658791542053, -1.7036486864089966, 1.2022966146469116, 2.0256834030151367, 0.15326490998268127, 0.5597053170204163, -1.2384495735168457, -0.6249357461929321, 1.3923242092132568, 0.3475455939769745, -0.5449373126029968, 0.1833351105451584, 0.7000798583030701, -0.2468116581439972, 0.9384763836860657, -0.1567886918783188, -0.2358240932226181], [-1.6930326223373413, -0.2840113341808319, -0.204311802983284, 0.12151971459388733, 1.0122554302215576, 0.5995315909385681, -0.49634894728660583, -1.5234962701797485, 0.5837892293930054, -0.7692293524742126, -0.18300706148147583, 0.9805194139480591, 0.2749481797218323, -1.5761675834655762, -1.3576446771621704, 0.8035827875137329, 0.7132003903388977, -0.1691712737083435, 0.2122107595205307, 0.45753875374794006, 1.2021864652633667, -0.5594030618667603, 0.7338692545890808, 0.9766713380813599, 0.29939067363739014, -1.1096214056015015, -0.2148919552564621, 1.9681860208511353, 0.14583152532577515, -0.215717613697052, -1.181923508644104, 0.734446108341217, -1.3127238750457764, 0.4432714283466339, -1.4468719959259033, 0.5030899047851562, -0.6095300912857056, 1.6152822971343994, 0.3013596534729004, -0.015542804263532162, -2.663435697555542, 0.6225888133049011, -1.2636975049972534, -0.1580103039741516, 0.8236801624298096, -0.8098501563072205, 0.41147857904434204, -1.9101594686508179, -0.11353928595781326, 0.3994866609573364, -0.47381359338760376, 2.5949904918670654, 0.27823200821876526, 0.8364174365997314, -0.05279164761304855, 1.577249526977539, -0.5185239911079407, 0.1904952973127365, 0.4357498288154602, 0.4457126557826996, -0.9824533462524414, -0.15648438036441803, 0.004572154022753239, -1.545030951499939, -0.7410544157028198, 0.6958776712417603, 0.6638062000274658, 0.7746741771697998, -0.13266946375370026, -0.224614217877388, -1.601552128791809, -1.5629469156265259, -0.42149364948272705, 0.5633347630500793, 0.9690154790878296, 0.5573760867118835, -1.4823030233383179, 0.40130117535591125, 2.108576774597168, 0.7465946078300476, -0.2691819667816162, 0.678006112575531, -0.2924456298351288, 0.173580601811409, -0.1819387525320053, -2.8736634254455566, 0.5286046266555786, -0.7835663557052612, -0.28985756635665894, -2.305626630783081, 0.6134675145149231, 1.7726976871490479, -0.5752930641174316, 1.7141833305358887, 0.8182504177093506, 0.9450668096542358, -1.1483879089355469, -1.7532365322113037, -0.1868003010749817, -1.0783473253250122, 0.14589087665081024, 0.7427688241004944, -0.5963737964630127, 0.14147132635116577, -0.17583486437797546, -0.09252360463142395, 0.8155333399772644, -0.9340975880622864, 1.3611005544662476, -0.42384329438209534, -0.24464046955108643, -0.7912949919700623, 0.27631402015686035, 0.6803326606750488, -0.5273322463035583, 0.20007413625717163, -0.436069518327713, -0.9398082494735718, -0.7919589281082153, 0.4143153429031372, 0.3691389858722687, 1.2419790029525757, 0.9201148748397827, 1.403600811958313, -0.3391488194465637, -1.7088685035705566, -0.226814404129982, -1.9015798568725586], [-0.5298659801483154, -1.2638142108917236, 1.0708353519439697, 0.678068995475769, 0.591341495513916, -0.6092159748077393, 2.3092029094696045, 0.2546653151512146, 0.5484122037887573, 0.8402906060218811, 0.590580940246582, 1.6159617900848389, 0.8640493750572205, 0.3291746973991394, -0.5320610404014587, 1.1043778657913208, 0.21915416419506073, -0.4572950005531311, 0.26826539635658264, 0.6610500812530518, 2.4503257274627686, -1.084044337272644, 0.13819871842861176, 0.40469303727149963, -2.7940242290496826, -0.05858107656240463, -0.08069489151239395, -0.4474303722381592, 0.02866603061556816, 0.3308177888393402, -0.4630982577800751, 0.08622682094573975, -1.009806513786316, 1.4716722965240479, -1.121018409729004, 0.6885011196136475, 1.002697229385376, 0.5911164879798889, 0.0009936258429661393, 0.7760423421859741, 0.7836624383926392, -1.4428826570510864, 1.0474048852920532, -0.09523560851812363, 0.62930828332901, 0.13985952734947205, -1.3794463872909546, 1.5333138704299927, -0.5257661938667297, -0.16313007473945618, -0.5993012189865112, 1.4551507234573364, 0.07491698861122131, -0.9465311765670776, 0.6956344842910767, 1.3798335790634155, -0.43925580382347107, -1.8135696649551392, 0.2724071443080902, -0.06517647951841354, 0.9246999025344849, 1.5260021686553955, -1.3908190727233887, -2.358639717102051, -0.4055866301059723, -0.24429070949554443, 1.5594866275787354, 0.6460145115852356, -0.8037464022636414, 1.7932895421981812, -0.5932008624076843, -0.8909602165222168, 0.13881845772266388, 0.3421792984008789, 0.501262366771698, -2.2934699058532715, 0.8750841617584229, 0.6648783087730408, 1.344260334968567, -0.47678491473197937, -1.5812698602676392, 1.2167953252792358, 0.6817980408668518, 0.3750058710575104, -0.5686945915222168, -0.6341385245323181, -0.831106424331665, 0.2724977433681488, -0.7164268493652344, -0.7236199975013733, 2.0576984882354736, 1.0292738676071167, -0.30109903216362, 1.2567663192749023, -1.1014623641967773, -0.4090781509876251, -0.9380471706390381, -1.5009247064590454, 0.46929728984832764, 3.0178592205047607, -1.3333708047866821, 0.2822514772415161, -1.003127098083496, -0.9591072797775269, -1.6791987419128418, -0.325827032327652, 1.7833242416381836, 1.6124258041381836, 0.29345133900642395, -1.0632712841033936, -0.5306796431541443, -0.9029290080070496, 1.4982218742370605, 1.168758511543274, -0.47812727093696594, 1.4774906635284424, 0.6512771844863892, 1.4880503416061401, -0.12426567077636719, 0.32657599449157715, -1.1242355108261108, -0.013437481597065926, 0.3921988904476166, 0.258629709482193, -0.38167792558670044, 0.15728230774402618, -2.2734546661376953, 0.11886659264564514], [0.27124762535095215, 0.16458895802497864, 0.11830365657806396, 0.14475080370903015, -0.026086296886205673, 1.967685580253601, 0.5304247140884399, 1.1550588607788086, -0.046782366931438446, -0.2522752285003662, 0.0016110822325572371, 0.6733793020248413, -0.5648553967475891, 0.6202268004417419, 0.7378517985343933, -0.15948572754859924, 1.759840965270996, 0.9184197783470154, 1.1331822872161865, -0.1764039397239685, -0.437399685382843, 0.5431005954742432, 0.3540765345096588, 0.6140134334564209, -2.7844858169555664, 1.2424591779708862, 0.6333993077278137, -0.7851446270942688, -0.984106183052063, 1.8707159757614136, -0.6589159965515137, 1.0285134315490723, -0.5265612602233887, -2.843095302581787, -0.09838738292455673, 1.766046166419983, -0.015250238589942455, 0.5715504288673401, 1.5208333730697632, -1.0578123331069946, 1.2496232986450195, 1.023224949836731, -2.11348295211792, -0.8616522550582886, -0.033636488020420074, -1.6541765928268433, -1.3710922002792358, 0.05670774728059769, 1.1793475151062012, 1.2510786056518555, -0.28172507882118225, 0.7472312450408936, -0.016948603093624115, -0.02019999362528324, 0.2553215026855469, 2.521524429321289, 0.11211135238409042, -2.1858937740325928, 0.10009867697954178, -0.3792932629585266, 0.39643043279647827, 0.34268462657928467, 1.346616506576538, -0.7570539712905884, 1.5742778778076172, -0.015488691627979279, 0.5135747194290161, -1.6580852270126343, 1.063354253768921, 0.19621393084526062, -0.41421082615852356, -0.2900272309780121, 0.42444127798080444, 0.541280210018158, 2.2982282638549805, -0.36497169733047485, 0.8170746564865112, -0.17884713411331177, 0.4689745008945465, 0.03544284775853157, 0.6081382632255554, 0.5253689885139465, 0.8662559986114502, 0.286380797624588, -0.25757989287376404, 1.1094517707824707, -0.7446637749671936, -0.39891517162323, -0.7125513553619385, -3.020338535308838, -0.04250255599617958, -0.615136981010437, 0.7579387426376343, 1.170265555381775, -0.14922845363616943, -2.0513525009155273, -0.00030335987685248256, 0.09683991968631744, -0.3751044273376465, 0.4789808690547943, 0.25651106238365173, 0.5542129874229431, -0.0310481209307909, 0.9195349812507629, -1.0942974090576172, -1.3002680540084839, -0.2790739834308624, -1.0338221788406372, -1.9169659614562988, -0.08273506909608841, -1.8661656379699707, 1.2569292783737183, 0.35211247205734253, 0.6584952473640442, -1.5536599159240723, 0.14008553326129913, -1.4705595970153809, -0.1453353315591812, 0.9317476153373718, 1.7465589046478271, 0.2620568871498108, -0.017921974882483482, 2.2395670413970947, 0.8379952311515808, 1.1963348388671875, -1.125133752822876, 1.3777987957000732, 1.574779987335205], [0.3140101134777069, -1.9488880634307861, 0.19119009375572205, -0.3135225176811218, 0.013328848406672478, 0.8237069845199585, -0.600845217704773, 1.432166337966919, -0.7763124108314514, -3.092212677001953, 0.5597324967384338, 1.5005199909210205, -0.5174193382263184, -1.9058939218521118, 0.02562924660742283, -1.474343180656433, 0.4204636514186859, 2.448268413543701, -0.7701936364173889, -0.24379833042621613, -0.6017109155654907, 0.3738599121570587, 0.9824236631393433, 0.0669800192117691, -1.252949595451355, 0.6241602897644043, -2.504606246948242, -0.9792425632476807, 1.8492872714996338, -0.6061496138572693, 0.4032345116138458, 1.1578108072280884, -0.63468998670578, -0.7650668621063232, -0.5978085398674011, -0.554667055606842, -0.9000483751296997, 0.4774154722690582, 0.7797410488128662, -1.1313376426696777, 0.42975518107414246, 0.07953602075576782, -1.096085548400879, -1.7255276441574097, 1.3296021223068237, -0.22176675498485565, 0.047741349786520004, -0.9780240654945374, 0.09488920122385025, 0.5825438499450684, 0.06718642264604568, 1.0002999305725098, -0.692963182926178, 1.3664759397506714, 0.71925950050354, 2.0129153728485107, -0.33720946311950684, -2.280890703201294, 0.3718617260456085, -0.6534611582756042, -0.16631561517715454, 1.2166895866394043, -1.2635297775268555, -0.5389942526817322, -0.9554802775382996, -0.684900164604187, -0.32325541973114014, -0.037276554852724075, -1.0200048685073853, 0.49303513765335083, 0.0704541876912117, -0.7025552988052368, -0.29867348074913025, 1.0298887491226196, 0.1559397578239441, -0.27199575304985046, -0.8988761901855469, -0.9645707607269287, -1.3598212003707886, -0.42949622869491577, -0.30921587347984314, -0.3474726974964142, -0.010648533701896667, 0.1638561338186264, -1.0327690839767456, 0.14668746292591095, -0.569065272808075, 1.2588456869125366, 0.4591081738471985, -0.6376036405563354, -0.48485395312309265, -0.2892301380634308, 1.100494146347046, -0.051493965089321136, 0.8679093718528748, 0.8010499477386475, -1.6345278024673462, 0.22022682428359985, 0.40077269077301025, 0.37726402282714844, -1.2712993621826172, 0.541605532169342, -0.5031201243400574, 1.623683214187622, 0.3490865230560303, 0.4491640627384186, 0.48835888504981995, -0.2970430552959442, -1.1080224514007568, -0.2433151751756668, -0.6507778167724609, -0.807210385799408, 0.5557493567466736, -0.04057659953832626, 0.17117054760456085, 0.6903958916664124, 1.5838385820388794, 0.6853949427604675, 0.5013180375099182, -0.5116056203842163, 0.16550257802009583, -0.4985698461532593, -1.0551577806472778, -0.40755730867385864, -0.7004652619361877, 0.7607021927833557, 1.0799355506896973, 0.9636618494987488], [-0.039811547845602036, 0.6031773686408997, 0.8640837073326111, -0.9951786398887634, -0.7166884541511536, -0.013569062575697899, -1.301872968673706, 1.5519627332687378, 0.5343130230903625, -0.17969374358654022, 0.734973132610321, 0.035390932112932205, 0.7378310561180115, 0.6115378737449646, -0.17525549232959747, 1.245815396308899, 0.8669117093086243, -0.2981087267398834, 0.36782166361808777, -0.6590025424957275, 1.3982514142990112, 0.6885584592819214, -1.1583874225616455, 1.9782804250717163, 1.2267762422561646, 0.7797016501426697, -0.1951105296611786, -0.5818999409675598, -1.8919827938079834, -0.7737171649932861, -0.7490222454071045, 0.6169694662094116, 0.9082669019699097, 0.7767816781997681, 1.2811224460601807, 0.450936496257782, 0.9648205637931824, 0.6869592070579529, -1.951704740524292, -1.5230860710144043, -1.3369531631469727, 1.3327784538269043, -0.9571335315704346, -0.47617092728614807, -0.09151721745729446, -0.51526939868927, 1.1358519792556763, -1.3205143213272095, -0.4389173090457916, -1.4645404815673828, 1.4785685539245605, -0.541120171546936, 0.9479551315307617, -0.6665824055671692, -0.6778622269630432, -0.1609019935131073, 0.24204599857330322, -0.17186449468135834, 0.35015273094177246, 0.9864622950553894, 0.6646367311477661, -0.13977006077766418, 0.15832625329494476, -0.63768470287323, 1.1283231973648071, -0.9280473589897156, -1.0033071041107178, -0.6384662985801697, 0.16914582252502441, 0.9043424129486084, -0.7312080264091492, -0.177341490983963, 0.9132991433143616, -1.187106728553772, 0.8427338004112244, -1.655990719795227, -1.8778656721115112, 2.930264949798584, -0.9201199412345886, -1.800764560699463, 0.14775873720645905, 0.7909804582595825, -1.2032825946807861, 0.08564560115337372, -1.721451997756958, 0.6850171089172363, 1.2413479089736938, 0.8977218866348267, -0.7474973201751709, -0.46802419424057007, -0.183253213763237, 0.9133831858634949, 0.7777805924415588, 0.9805783629417419, 0.2956434190273285, 0.027549153193831444, 1.2070026397705078, -0.162444606423378, 1.4987772703170776, 0.41463667154312134, 1.5189340114593506, 0.7128019332885742, -0.9976335763931274, 1.039594054222107, 0.29243871569633484, -0.2179969698190689, -0.23800531029701233, 2.160687208175659, -1.1773236989974976, -0.5241000652313232, 1.0642995834350586, 0.7988548874855042, -1.0629128217697144, -0.23112216591835022, 0.4013550281524658, 0.18131299316883087, -0.37071794271469116, 0.05131815746426582, -1.1068099737167358, -0.2097885012626648, -2.5652174949645996, 2.346649169921875, -0.21529269218444824, 0.3994153141975403, -0.18958856165409088, 2.1368885040283203, -0.8273922204971313, 0.8211578726768494], [0.6670554280281067, 0.176240012049675, 0.8220310807228088, -0.3732798993587494, 1.490419626235962, 1.8416759967803955, 0.30500394105911255, 0.4332629144191742, -1.439044713973999, -0.35799676179885864, 0.0034435598645359278, 0.5950583219528198, -1.6340293884277344, -0.011157427914440632, -2.4394445419311523, 0.488042950630188, 0.46302780508995056, 0.15425187349319458, 0.42260247468948364, -1.659445881843567, -0.2338877171278, 0.3871999680995941, 1.3228753805160522, 2.1171209812164307, 0.7332896590232849, 1.7862651348114014, 1.2702075242996216, 0.6945797204971313, 1.0332589149475098, -0.9781078100204468, 0.2462375909090042, 0.06757329404354095, 0.69500732421875, 1.5099868774414062, 0.593184769153595, -1.051659107208252, -0.08896876126527786, -0.3425748348236084, -0.6530959010124207, -0.5359898209571838, 0.22003938257694244, -0.1796654313802719, -1.955209493637085, 1.3998421430587769, -1.2796900272369385, 1.7608815431594849, -1.5601199865341187, 0.006904347334057093, -0.9884458780288696, 0.9945738315582275, 1.250908613204956, 0.01850745640695095, 0.03562324494123459, 2.0945606231689453, 0.07874242961406708, -0.7714827656745911, -0.050901755690574646, -0.12075352668762207, 0.3702087700366974, -0.08694054931402206, 0.20580126345157623, -1.300197720527649, 1.0816065073013306, 0.17311811447143555, 2.1044862270355225, 1.5205978155136108, -0.4068602919578552, 1.4184205532073975, 0.9737269878387451, 1.1607093811035156, 0.9585371017456055, -1.4595346450805664, -0.21591177582740784, -1.5974385738372803, 0.6217238306999207, -0.16578270494937897, 0.7495942711830139, 0.25203362107276917, 0.17791037261486053, 1.5909006595611572, 1.0651130676269531, 0.9786630272865295, 0.1465236246585846, 1.2368133068084717, 0.08836926519870758, -0.8545131087303162, 1.3803366422653198, 0.4081684947013855, 0.04562890902161598, 0.02690877765417099, -1.7759099006652832, 1.6885936260223389, 0.17952242493629456, 0.8997268676757812, 0.08534654974937439, 0.7674263119697571, -1.091139554977417, 0.24905353784561157, 1.6003011465072632, -1.2163041830062866, 1.0921152830123901, 2.0061404705047607, 1.188952922821045, 1.2752574682235718, 0.02330322004854679, -2.336819648742676, -0.4237753450870514, -0.2432614266872406, -1.8549211025238037, -0.07388681173324585, 0.5024368762969971, -0.8585633635520935, -0.35643815994262695, 0.5336236953735352, 0.7505236268043518, -1.1824191808700562, -1.1933165788650513, -0.17999650537967682, 2.131155014038086, 0.706047773361206, 1.3645741939544678, 0.5900108814239502, -0.43031659722328186, 0.9598392844200134, -0.4800863265991211, -0.3208567798137665, 0.3570536673069, 1.0227758884429932], [1.167983889579773, 1.2400037050247192, 0.8728070855140686, 0.11082079261541367, 1.793416976928711, 0.6780964732170105, -1.3243283033370972, 1.1370943784713745, 1.5292738676071167, -0.1536983996629715, 0.2814081907272339, -0.9590885639190674, 0.3397965431213379, -0.5589616894721985, -0.905406653881073, -0.5674319863319397, -1.0970814228057861, -1.2782737016677856, 0.1808834969997406, 0.26162686944007874, -0.7587897777557373, 2.9490089416503906, -0.1778152585029602, -0.04473816603422165, 0.24809975922107697, -0.30867892503738403, 1.3830831050872803, 0.33648592233657837, 0.5966347455978394, -0.3898142874240875, -0.2668660581111908, 1.0301066637039185, 1.319941520690918, 0.04551123455166817, -0.2458420842885971, 1.8212077617645264, 0.764582097530365, -0.22259819507598877, -0.7448168396949768, 1.1463044881820679, -0.8577705025672913, -2.5024547576904297, 0.4937904477119446, 0.7848634719848633, -0.400693416595459, 1.876893401145935, -0.9268414974212646, -1.947005271911621, 1.432401180267334, 0.11399629712104797, -0.441618412733078, 0.30435246229171753, 1.5613410472869873, -1.5757298469543457, -0.7730645537376404, -0.09024225175380707, 0.8273404240608215, -0.3833325505256653, 0.873692512512207, 0.877102255821228, 0.2851864695549011, -0.39342421293258667, -0.5309209227561951, -0.16439363360404968, -1.063632845878601, 0.675607442855835, -1.7411712408065796, -0.6184204816818237, 0.16229335963726044, -0.2172277569770813, -0.37119150161743164, -1.8910084962844849, 1.1493747234344482, -1.6194850206375122, 0.7174413204193115, -1.0839403867721558, -0.605688750743866, 0.7524448037147522, 0.4300335943698883, 1.327245831489563, 1.614662766456604, -0.21967057883739471, -0.9137557148933411, 0.07751194387674332, 0.1270802915096283, -0.0908280536532402, -0.3181077837944031, -0.5083820819854736, 1.0840528011322021, 0.2652485966682434, -1.4281408786773682, -0.08134808391332626, -0.2030377984046936, 0.20797099173069, -0.49492955207824707, 1.526738166809082, 0.30367380380630493, -0.34650543332099915, 0.1461450606584549, 0.503956139087677, -1.6469712257385254, -0.29450902342796326, -0.5800262689590454, 1.6748576164245605, 0.4197002649307251, 0.7455191016197205, -0.022677304223179817, 0.9483871459960938, 2.04267954826355, -1.4589977264404297, 1.6665687561035156, 0.04970265179872513, -1.2721596956253052, -0.13088490068912506, 0.24502673745155334, -0.23491743206977844, 1.4721693992614746, 0.2707177698612213, 0.28463616967201233, 1.8591276407241821, -0.6409350037574768, -0.6891776919364929, -1.4094330072402954, 1.2164922952651978, -0.3938674032688141, 1.33449125289917, 0.722285807132721, -0.6441330313682556], [0.9003873467445374, -0.7068896293640137, 0.9660107493400574, -0.45947590470314026, 0.6689139604568481, 0.9965317249298096, 0.55604088306427, 0.35440316796302795, 2.3201234340667725, -0.3871070444583893, -0.7997035384178162, 1.567651629447937, 0.1751391589641571, 0.5500297546386719, -0.5746319890022278, 1.6663105487823486, 1.1910438537597656, -1.3486154079437256, 1.1772140264511108, -1.6433511972427368, -0.21067634224891663, 0.5992363691329956, -0.13090908527374268, -0.5254160165786743, 2.319521903991699, -0.5951361060142517, 0.3398149609565735, -1.8364654779434204, -1.0785678625106812, -0.9542778730392456, 0.12814714014530182, -2.3225231170654297, -0.019421828910708427, 1.2114180326461792, 0.6279762387275696, -0.7836113572120667, 0.5720099806785583, 0.24476683139801025, 0.7900460958480835, -1.1892695426940918, 0.4474213719367981, 0.8301500678062439, -0.11774498969316483, -0.4108138978481293, 0.5256066918373108, -1.7991162538528442, 2.4941658973693848, -1.8601106405258179, 0.779097318649292, -0.7723904848098755, 0.5830979347229004, -0.44086432456970215, 0.7543142437934875, -1.4055392742156982, -0.6065409183502197, 0.2435581237077713, 0.54616779088974, 0.05455424636602402, -0.45419344305992126, -0.3688097298145294, -1.0727365016937256, 0.5683995485305786, 1.5987318754196167, 0.9796847701072693, 0.6756736636161804, 0.0902252048254013, -1.1468188762664795, -0.574622631072998, 0.1898428052663803, -1.3643919229507446, 0.9490271806716919, -1.177268624305725, 0.26968303322792053, 0.2517106235027313, -2.0841495990753174, -0.052256807684898376, 0.13335637748241425, -0.8212475180625916, 0.3431771993637085, 2.1705498695373535, 0.6415781378746033, 0.12136471271514893, -1.4958933591842651, -1.5569789409637451, 0.055295102298259735, 0.0040426854975521564, -0.776913046836853, 1.6685442924499512, 1.2907204627990723, -1.4966225624084473, 0.5630773901939392, -1.2858963012695312, -0.6327120065689087, 0.007842021994292736, -0.46672728657722473, -1.6688830852508545, 0.6502898335456848, -2.7879717350006104, 1.6274049282073975, -0.858634352684021, 0.00898117758333683, 0.7217804193496704, -0.21250073611736298, -0.4192996621131897, 1.738451361656189, 0.48935002088546753, 0.8832899332046509, 0.34145689010620117, -0.1426374465227127, 0.4527502954006195, -0.09191180020570755, -0.6753582954406738, -1.057665228843689, -0.3352360129356384, 2.380831241607666, -1.1739721298217773, 1.2185218334197998, 0.2364896684885025, -0.8467878699302673, 0.9622728824615479, 0.8892816305160522, 1.0236730575561523, -0.679718017578125, 1.592808723449707, 0.8184959292411804, -1.7965620756149292, 0.061979152262210846, 0.9881117343902588], [-1.6533353328704834, -1.4602019786834717, -1.7416554689407349, 0.3237358629703522, -1.1830437183380127, -0.21639588475227356, -1.8230606317520142, -0.38666555285453796, -0.9256870150566101, 0.4028478264808655, -0.4707837402820587, -1.149233102798462, 0.2419915795326233, 0.5321269035339355, 0.40410083532333374, -1.9987587928771973, -1.217814564704895, 1.795422911643982, 0.37008795142173767, -1.2119319438934326, 0.20832857489585876, 0.39942467212677, -0.31514689326286316, -0.19558876752853394, -1.251085638999939, -1.1697531938552856, -1.9679166078567505, -0.5771827697753906, 0.4494251608848572, -0.9999982118606567, 0.7820863723754883, 0.41846534609794617, -0.9809589982032776, 1.1848173141479492, 0.35520827770233154, 1.168443202972412, 0.34925422072410583, 2.7676143646240234, -0.9086522459983826, -0.5166224837303162, -0.027761364355683327, -2.1817522048950195, -0.20957206189632416, -0.8453063368797302, 0.6366986036300659, 1.0923771858215332, 0.6348935961723328, 0.23914258182048798, 0.3929188847541809, -0.4301726818084717, -0.8546566963195801, -0.837471604347229, -1.185516119003296, -0.4344969093799591, 0.1830286979675293, -0.10323646664619446, -1.1137346029281616, 0.18066523969173431, -0.8690162301063538, -0.3374553322792053, 1.3035743236541748, 0.8803259134292603, 1.1016175746917725, 0.08001567423343658, -0.5126320719718933, -1.6827813386917114, -0.23988334834575653, 0.016657143831253052, 1.2388542890548706, 2.0996856689453125, 0.3490723669528961, -1.798629641532898, -0.6911231875419617, 0.3151494562625885, 1.4290422201156616, -0.12783755362033844, 0.7164369821548462, -0.0375579334795475, 0.7811807990074158, 0.7617396116256714, -0.1589146852493286, 0.6904261708259583, 0.6170729398727417, -0.15267552435398102, -2.0607781410217285, -0.26944154500961304, 0.9406092762947083, 0.7947021126747131, 0.7366787791252136, -0.5032356381416321, 0.7733919620513916, 1.3661528825759888, -0.03622807189822197, 0.19856393337249756, 0.7178195118904114, 0.5621362328529358, 0.26762494444847107, -0.6830846071243286, -0.9915497899055481, 0.2776185870170593, 0.3862578868865967, 0.07164192199707031, 1.2655483484268188, -0.02226411923766136, -0.1121034100651741, -0.3842525780200958, -0.49549436569213867, -1.2921473979949951, -0.4776725471019745, -0.14212936162948608, 0.6598790287971497, -1.9305437803268433, 1.0789766311645508, 0.09222660213708878, 0.9787183403968811, -0.5798619985580444, 0.17130771279335022, 0.17541125416755676, -0.5186890959739685, -0.21932344138622284, -0.2567865550518036, -0.35609084367752075, 1.4774903059005737, -0.31460562348365784, 0.6366395354270935, -0.845372200012207, 0.06423897296190262, -0.23841093480587006], [0.2632085382938385, 0.3070344924926758, -0.9003698229789734, -1.7731274366378784, 0.11366105079650879, -0.5718843936920166, -1.7713145017623901, -1.0228995084762573, -1.0447719097137451, -0.4160110056400299, -0.016727743670344353, -0.646571159362793, 1.5053753852844238, 0.5040534734725952, 0.8720205426216125, -0.05064714327454567, -1.6465190649032593, 0.9769951701164246, 1.6160911321640015, -0.12677690386772156, -1.545426368713379, 0.3444035053253174, 0.7151625752449036, -3.0939931869506836, -1.3619672060012817, 0.49962061643600464, 0.26170119643211365, 0.6799102425575256, -0.7575814127922058, -0.09271416068077087, 0.2640286087989807, -0.13547855615615845, 0.9675052165985107, 1.1900912523269653, 0.6276452541351318, -0.09921993315219879, 0.0326094850897789, -0.22219257056713104, -0.033755674958229065, 0.6758436560630798, 1.3647966384887695, 0.6535900831222534, -0.41275832056999207, 1.0063462257385254, 0.7148202061653137, 1.101693868637085, -2.1417770385742188, 0.9450786113739014, -1.356092929840088, -0.07239269465208054, 0.19374744594097137, -0.9272586703300476, 0.6146535873413086, 1.634464979171753, -1.5651484727859497, 0.7437136173248291, 2.1410040855407715, -0.8091065883636475, -0.39291343092918396, -0.8113666772842407, -0.7948471307754517, -0.8087158203125, -0.4559457004070282, -0.23410166800022125, 0.5700506567955017, -0.12262942641973495, -0.7516908049583435, -0.5522962808609009, -0.38950878381729126, -1.8811286687850952, 1.0989426374435425, -0.4112260043621063, -0.8211973905563354, -0.04744937643408775, -1.1528321504592896, -0.19077923893928528, 0.11491918563842773, -0.1391848623752594, -1.1382471323013306, 0.7225068211555481, -0.19389256834983826, 0.7932407259941101, -1.2604464292526245, -0.587956428527832, -0.36439448595046997, -0.48466384410858154, 0.9707752466201782, -0.33753570914268494, 2.3066279888153076, 2.4312214851379395, 2.2859091758728027, -1.04962956905365, -2.3461835384368896, -1.2965683937072754, -1.3498324155807495, 0.4178873300552368, -1.3834527730941772, 1.6930407285690308, 0.7678627371788025, 0.3402889668941498, -0.16580364108085632, 1.172284483909607, 0.7071788907051086, 1.1637835502624512, 1.5721200704574585, -1.1481094360351562, 0.3560290038585663, -0.9633552432060242, 0.5684093832969666, 0.6283484697341919, 1.2714462280273438, 0.010106203146278858, -0.4633101224899292, -0.18541020154953003, 0.8562650680541992, -1.0852503776550293, -0.6146815419197083, -1.1037981510162354, 0.5177047848701477, 0.4473923444747925, -0.9094889760017395, 0.4672684371471405, 1.8215970993041992, 0.057746510952711105, 0.07614292949438095, 2.184782028198242, -1.1481002569198608, 0.2950226068496704], [0.5394085049629211, -2.3969452381134033, -0.346049040555954, -1.6918160915374756, 0.022302409633994102, -0.3134192228317261, 1.7472913265228271, -0.5467780828475952, -2.0574791431427, 1.0491127967834473, 1.5726701021194458, 0.007268615998327732, -0.6977748870849609, 0.1449267417192459, 0.41879990696907043, -0.05005645379424095, -0.07730825990438461, 0.5061310529708862, 1.4381884336471558, 1.7035276889801025, 1.707472562789917, 1.9734963178634644, 0.2642926573753357, -1.0013560056686401, 0.3647702932357788, -0.27899065613746643, -0.8700121641159058, -0.8818623423576355, -0.062142521142959595, 0.4097520411014557, -0.34951168298721313, -0.3423663079738617, -0.1564975380897522, 0.4883776009082794, -0.35016441345214844, -2.2098212242126465, 1.7042791843414307, 0.3836303949356079, 0.6911745071411133, 0.7943276166915894, 0.015778955072164536, 0.5861324667930603, -0.36225658655166626, 2.3749327659606934, -0.8035104870796204, 1.734454870223999, -0.1374453753232956, -1.3536862134933472, -1.9125417470932007, 0.06854409724473953, -2.071833848953247, 0.25186869502067566, 1.51982843875885, 1.3510072231292725, 0.5429275631904602, 3.2342395782470703, 0.7839298248291016, -0.17015719413757324, 0.5731907486915588, -0.4185318052768707, 0.2406935840845108, -0.5762285590171814, -0.78608638048172, 0.051906850188970566, -1.2581368684768677, -1.4168070554733276, -1.0982003211975098, 0.34494534134864807, -1.1745737791061401, -0.8407520055770874, 0.6580320000648499, -1.457546353340149, 0.8845716118812561, -0.8369802832603455, -0.8013685345649719, -0.0024289304856210947, 0.10661010444164276, 0.7509618401527405, 0.6132997274398804, 1.1314527988433838, 3.1594486236572266, -0.8475711345672607, 0.15657897293567657, 0.006533011794090271, 0.5023981928825378, -1.4584156274795532, -0.5861316323280334, 1.563982605934143, -0.5970779657363892, -0.3221726417541504, -1.4166699647903442, -0.15515488386154175, -0.35658732056617737, -0.39959946274757385, 1.710015892982483, 0.9837565422058105, -0.9022293090820312, -1.0723997354507446, 1.420004963874817, 0.4694691002368927, -1.8641396760940552, 0.8212056159973145, 2.201352119445801, -0.7393127083778381, -1.5246362686157227, -1.9605624675750732, -0.4119507968425751, -0.06326530873775482, 0.007366941310465336, -0.5217618942260742, 0.6024919152259827, 0.3506562113761902, 0.8054545521736145, -0.07937243580818176, -0.13712048530578613, -0.3888816237449646, -0.3143916130065918, -0.46081987023353577, 0.3032538890838623, -2.2426419258117676, -0.005326972808688879, -0.05277285352349281, 0.17818424105644226, 0.7821739912033081, 0.9502089619636536, -2.221076250076294, -0.7651772499084473, -1.5838836431503296], [1.5744551420211792, 0.326322078704834, -0.2955315113067627, 1.0506641864776611, -0.777914822101593, 0.12964555621147156, 0.8184725642204285, 0.8449187278747559, 0.6136152148246765, 1.0202723741531372, -0.016348714008927345, 0.2994694709777832, 0.35547879338264465, 0.2882065176963806, 1.2929667234420776, -0.1131812185049057, -1.0843278169631958, 0.8008168935775757, 0.6964081525802612, 0.49252834916114807, -0.14291511476039886, -0.24914225935935974, 0.5907509922981262, -0.5734383463859558, -0.017839858308434486, 0.9078859090805054, 0.4768458902835846, 0.8488999605178833, -1.1909645795822144, -0.9295316338539124, 1.024031162261963, 1.222578525543213, 0.04963766410946846, 0.855888307094574, 0.17614129185676575, 1.8867439031600952, -1.5129255056381226, 2.2102649211883545, -0.5030085444450378, -0.4159446060657501, 0.42286738753318787, 1.0079007148742676, 0.792568564414978, -0.2735908329486847, -1.4145864248275757, 0.8955830931663513, -0.03247992321848869, -0.02649000473320484, 3.0754220485687256, -0.8655933737754822, 1.0836329460144043, 0.1605217158794403, -1.265743374824524, 0.4700445532798767, -0.5371273756027222, 0.31239771842956543, -0.2833155691623688, -0.892737627029419, 0.15887373685836792, -0.6098835468292236, -1.5016121864318848, -0.2650240957736969, -0.8309324979782104, 0.2510136365890503, -0.06244318187236786, 0.09459275007247925, 0.5992065668106079, 0.31877362728118896, 1.8251426219940186, -0.8604645729064941, -0.09755405783653259, -1.1990928649902344, -1.188550591468811, -0.1071336418390274, 2.2109100818634033, 0.5822893381118774, 0.505736768245697, 0.4528197646141052, -0.769309401512146, -0.1027049571275711, 1.3683375120162964, -0.28034859895706177, 1.0543813705444336, -0.6420798897743225, -1.6398367881774902, -1.103550910949707, 1.3768750429153442, -0.8810907602310181, -0.2859458923339844, -0.16046380996704102, 2.121628761291504, 1.8258854150772095, -1.2579283714294434, -1.0631098747253418, 0.5377939939498901, 1.1299140453338623, -1.385853886604309, -0.10115911066532135, -0.636022686958313, -0.3038993775844574, -1.3902441263198853, 0.36162567138671875, -1.5280487537384033, 0.29571807384490967, 0.3578559458255768, 0.04101342707872391, -0.7404640316963196, 1.8155242204666138, 1.2528038024902344, 0.34709206223487854, -0.900616466999054, -1.962794542312622, -0.18948014080524445, -1.2797596454620361, -1.1432874202728271, 0.7578368186950684, 1.1210583448410034, 0.5735321640968323, 0.07320179790258408, 1.5728758573532104, -0.6512284874916077, 0.6538901329040527, -0.09573721885681152, 1.4410821199417114, -0.9658191204071045, 1.9099727869033813, 0.0417441762983799, 0.3705310821533203], [-0.30450573563575745, 0.29794418811798096, 0.14390112459659576, -1.8932807445526123, -0.4430210292339325, 1.1893962621688843, -0.3486909866333008, -0.14356543123722076, 0.38229623436927795, -0.9587192535400391, -1.2266643047332764, -0.5139713883399963, -0.4762570261955261, 2.061560869216919, 2.5992279052734375, 0.7560071349143982, -0.3706015348434448, -0.7544931173324585, -0.9699856638908386, -0.73212069272995, 0.5577508211135864, 0.16669277846813202, -0.5951476097106934, -0.25519782304763794, 0.9512012600898743, 1.1944093704223633, 1.4406991004943848, 2.167651891708374, -0.8763169050216675, 0.5990950465202332, -0.9859153032302856, -0.7693026065826416, -1.3141324520111084, 0.535720705986023, 0.08479390293359756, 0.6378691792488098, 1.3362524509429932, 1.4927997589111328, -0.47234195470809937, -0.23518656194210052, -1.2724472284317017, 1.10409677028656, 0.4705972671508789, -0.044582549482584, 0.43300291895866394, 1.6467424631118774, 0.669903039932251, -0.734072744846344, 1.8309054374694824, 0.04223038628697395, -0.2752455472946167, 0.12231907993555069, 1.674232840538025, 0.3497849106788635, -0.421130895614624, -0.8777462840080261, -0.1557343602180481, 1.0408704280853271, -1.3301321268081665, -1.6228222846984863, 0.15477916598320007, -2.0072431564331055, -0.540059506893158, 1.1323559284210205, 0.09076395630836487, -0.5479495525360107, -0.4876367449760437, -1.0936918258666992, -1.2493314743041992, 0.3804599642753601, 0.43485385179519653, 0.4517175257205963, -0.550821840763092, -0.49837470054626465, 0.3851347267627716, 0.36246457695961, 0.5010454058647156, 1.831575632095337, -0.9826905727386475, 2.384458065032959, -0.1614135354757309, 0.7848504185676575, 2.150204658508301, -1.7666947841644287, -0.40431728959083557, 1.2862476110458374, -0.8431171774864197, -0.6608656048774719, -0.08144226670265198, 1.1673699617385864, -0.33645087480545044, -0.32280951738357544, -1.1819608211517334, 1.9838005304336548, -0.9008405804634094, -1.1345638036727905, -0.1163523942232132, 0.25148653984069824, 0.11739913374185562, -0.128273606300354, -1.9200161695480347, 1.2695722579956055, 1.5546644926071167, 1.1556179523468018, 0.8975890874862671, -0.09622253477573395, -0.35817036032676697, 0.6882736086845398, 0.976237416267395, 0.8543545603752136, -1.28016996383667, 1.0238933563232422, -1.4011189937591553, 1.5309138298034668, 1.7506113052368164, -1.8233013153076172, 2.222055196762085, -1.252905249595642, -0.0044790152460336685, 0.6845923066139221, -0.6253058910369873, -0.3300445079803467, -1.715815782546997, -0.007593974936753511, -0.13624213635921478, -0.09226715564727783, 0.7551908493041992, 1.0171207189559937], [0.23010191321372986, -1.6687257289886475, 0.997017502784729, 1.077807903289795, 0.8351292610168457, -0.6191691160202026, -1.274810552597046, -0.8480709791183472, 0.9609830379486084, 0.6937135457992554, -0.9399760365486145, 0.9691116809844971, 1.4772087335586548, 0.5209885239601135, 0.5615431666374207, -0.17573830485343933, -1.0684418678283691, -0.5644872784614563, 1.1583771705627441, 1.186293601989746, 1.2434697151184082, 0.7102512121200562, -1.2808741331100464, -0.7852211594581604, -1.4392637014389038, -0.35205402970314026, -1.649383544921875, -0.7347331643104553, 1.046146035194397, -0.6777902245521545, -0.05733728036284447, -0.06339731067419052, 0.27182725071907043, 0.4881806969642639, 0.7609742283821106, -0.8568674325942993, 0.9225348234176636, 1.555106520652771, -1.016638159751892, -1.058915138244629, -0.9255058765411377, 1.3136688470840454, -0.43730229139328003, 0.17679093778133392, -0.18856358528137207, -0.8018760085105896, 0.2506301701068878, -1.475175380706787, 0.8105257153511047, -0.9730983376502991, 0.5759010910987854, 0.9200031161308289, -0.803188145160675, -1.683448076248169, -0.49234631657600403, 1.4629642963409424, 0.6908409595489502, -0.8185417056083679, 0.9340969324111938, 1.0817514657974243, -0.5432102084159851, -1.0280702114105225, 0.8171625137329102, 0.6526238918304443, 1.1232693195343018, -0.4973348379135132, 0.04848852381110191, -1.6813853979110718, 1.1430577039718628, -1.3718085289001465, 1.4246201515197754, -0.14741453528404236, 2.266279458999634, 1.7056386470794678, 0.951178789138794, -1.337372899055481, 0.26863405108451843, 1.1481027603149414, 3.4769701957702637, -0.1454010307788849, -0.6293060183525085, -2.7470576763153076, -2.5228617191314697, 0.46851587295532227, -0.6699427962303162, 0.9509104490280151, -0.8195191621780396, -0.3034372925758362, 0.25456592440605164, 0.3666161894798279, 0.20281916856765747, 0.7550346255302429, 0.254512757062912, 0.787360429763794, 0.7196658253669739, -0.8521923422813416, 0.05234192684292793, -0.7696577906608582, 0.6773847937583923, -0.6936976313591003, -0.3823626637458801, 0.4868534505367279, 0.6902714967727661, 0.33391132950782776, -2.085865020751953, -1.4746569395065308, 0.7380182147026062, -0.1438959240913391, 1.4086170196533203, 0.37287068367004395, -2.0489399433135986, -1.193385362625122, 1.3131775856018066, 1.0243245363235474, -0.30561721324920654, -1.4451385736465454, -0.13455522060394287, -0.8467423319816589, 1.6773970127105713, 0.6883530616760254, 1.2096308469772339, 0.7750425934791565, -0.27620548009872437, 0.8531391620635986, 0.12111376225948334, -0.8782044053077698, 0.4965544641017914, 0.07636857777833939], [-1.5746400356292725, 0.11571560055017471, 0.9139714241027832, -0.07835329324007034, -0.41700008511543274, -1.00569748878479, -0.6253987550735474, 2.4224932193756104, 0.3916538953781128, 0.4322451949119568, -1.8049670457839966, 0.3472154438495636, -0.8621456623077393, -0.310051292181015, -0.8443114161491394, -0.7400732040405273, -1.0683550834655762, -1.074454665184021, 0.739172637462616, 0.0014518112875521183, 1.4725528955459595, 0.21599049866199493, 0.3555358648300171, 1.2746026515960693, -1.0296822786331177, -0.20269571244716644, 0.7140375375747681, -1.751621127128601, -1.1674888134002686, -0.06407657265663147, 0.4253961145877838, -0.67216557264328, -1.2305233478546143, -0.04808228835463524, 1.5934488773345947, 0.9993169903755188, 0.1216580718755722, -0.3286643922328949, -0.8516234159469604, 0.804190456867218, -0.11200138926506042, -1.7202903032302856, 0.5833657383918762, 0.8521655201911926, 1.453467845916748, 1.37328040599823, -0.2362101972103119, -0.48839274048805237, 0.6419246196746826, -1.9874396324157715, -0.2758593261241913, -0.8971683979034424, 0.43507304787635803, 1.7962586879730225, -0.4179791212081909, -0.5748420357704163, -0.4063885807991028, -0.5225121974945068, 0.058638427406549454, 2.231076955795288, 0.9445862174034119, 0.26335132122039795, 0.19592680037021637, -0.35226643085479736, 0.6643351912498474, -1.1845391988754272, 1.0824376344680786, 0.34013858437538147, 0.1946234107017517, -1.3529980182647705, 1.0381981134414673, 0.4695356786251068, 0.08584363758563995, -0.32253846526145935, 0.49306654930114746, 1.9650532007217407, 0.26732078194618225, 0.47964242100715637, -2.023160696029663, -0.018241411074995995, 1.4380700588226318, -1.2479047775268555, 0.24226883053779602, -1.9418718814849854, 1.7698100805282593, 0.8298547863960266, 0.20263977348804474, -1.6666678190231323, 0.9524551630020142, 1.641109585762024, -0.6926088333129883, 0.782387912273407, 0.29500314593315125, -0.3489394783973694, -1.4731197357177734, 0.3644426763057709, -0.17681224644184113, -0.5854448676109314, -0.6380172967910767, -0.881285548210144, -1.750284194946289, -0.9019474387168884, 1.3525357246398926, -1.9151262044906616, 1.2048100233078003, 0.8506922721862793, 1.1609433889389038, 0.6255161762237549, 1.1847500801086426, -0.8184753060340881, 1.385986328125, 0.21746350824832916, -1.2798064947128296, 0.8128321766853333, -0.9022789001464844, 2.3148505687713623, 1.873016357421875, 0.8119331002235413, 0.9616644382476807, 0.7447817921638489, -0.23364204168319702, -0.5087809562683105, -0.32728034257888794, -1.3097929954528809, 1.4410400390625, 0.45840924978256226, -0.7636802196502686, 1.7062283754348755], [-1.5217863321304321, 0.9737792015075684, -0.8636108040809631, 2.1973776817321777, -1.2882654666900635, -1.3497025966644287, 0.34795862436294556, 0.2717815041542053, 0.4177095890045166, 0.6118646860122681, -0.47779637575149536, -1.1351944208145142, -0.015610642731189728, 1.6331958770751953, -0.5392769575119019, -0.6756677031517029, 0.8088034987449646, -0.3935442566871643, -1.6172406673431396, -1.933736801147461, -0.9472312927246094, 0.8348308205604553, 1.016527771949768, 0.24092429876327515, -1.1586239337921143, -0.6766147017478943, 1.4590259790420532, 1.6829911470413208, 0.6545968651771545, 0.0025502718053758144, -0.16030475497245789, -0.6162059307098389, 0.0347440131008625, -1.7591947317123413, -0.80122971534729, -0.2885020077228546, 1.1435651779174805, -1.209040880203247, 0.468587726354599, -1.2361916303634644, 0.6266922950744629, -1.8298872709274292, 0.47426068782806396, 0.2729892432689667, 1.153691053390503, -1.224288821220398, -0.8136333227157593, 0.338199645280838, -0.16753292083740234, -0.8433467745780945, -0.01754923164844513, 0.13847312331199646, -1.5626500844955444, -0.1655423790216446, 1.837568759918213, -2.229172468185425, -0.560771644115448, -0.2410982996225357, -0.09828058630228043, 0.139382004737854, 0.6683362722396851, -0.3829216957092285, -0.40779319405555725, 1.3090397119522095, 0.374514102935791, -0.7698658108711243, -0.3613017499446869, -0.27665823698043823, 0.7054963111877441, -1.065793752670288, -2.0393199920654297, 0.33928608894348145, -0.9142195582389832, -1.5752863883972168, 1.4975472688674927, 0.09771372377872467, -1.7541617155075073, 0.7372218370437622, 0.18621183931827545, -0.3980032205581665, 0.13207575678825378, 0.23878948390483856, 1.5940889120101929, 0.30975541472435, 0.7122834920883179, 0.4339836537837982, 0.239213228225708, 1.7842389345169067, 0.5889021158218384, -1.1250264644622803, 0.07645749300718307, 1.2240626811981201, 1.7685168981552124, -1.2324042320251465, 1.2640067338943481, 1.4036649465560913, 0.30639591813087463, 0.6672641038894653, -0.8303453326225281, 1.6579290628433228, -0.439087450504303, -1.0653235912322998, -0.07949218153953552, -0.6083402633666992, -0.37336820363998413, -0.4598625600337982, -0.5733242630958557, 0.26304903626441956, -0.2922143340110779, -0.380429744720459, -0.23923546075820923, -0.4775873124599457, 1.0476934909820557, 0.28099584579467773, 0.36327674984931946, -0.11170625686645508, 0.574432373046875, -2.18257999420166, 0.04248204454779625, 1.4522161483764648, -0.28956255316734314, -0.1850472390651703, 0.793393075466156, -0.027049293741583824, -2.1848838329315186, 0.39947396516799927, -1.7509125471115112, 1.2409669160842896], [1.000726342201233, -0.24005872011184692, 0.13845527172088623, -0.3621155023574829, -0.9228251576423645, -0.24074995517730713, 0.9910323023796082, -0.2795618176460266, 0.40985068678855896, -0.976173460483551, 1.4255450963974, 0.6365911960601807, 1.2398643493652344, -1.0057636499404907, 0.39931201934814453, -0.06450306624174118, 0.04086737707257271, -1.0394402742385864, 0.33998236060142517, 0.32542189955711365, 0.36772555112838745, 0.6115315556526184, 1.9916586875915527, -0.9209679365158081, 0.43579623103141785, 0.2211899757385254, -0.2959357500076294, -0.5882900953292847, -1.0199705362319946, 1.128003478050232, -0.5774866938591003, 0.12038545310497284, -0.1452924609184265, -0.3450075089931488, -0.12329386174678802, 0.3284424841403961, 1.1542742252349854, -0.22431224584579468, 0.44906172156333923, 0.6348806023597717, 0.592236340045929, -0.11068940162658691, -0.9766237139701843, -0.6322082877159119, -0.3158092200756073, -0.8920300006866455, 0.5976595282554626, 0.764268159866333, -0.7972515821456909, -2.0737340450286865, -0.03179575875401497, 0.7519746422767639, -0.22609122097492218, -0.29576584696769714, -0.9534205198287964, 1.1866841316223145, -0.11100085079669952, -1.2990490198135376, 0.4587451219558716, 0.6714625358581543, 1.6710519790649414, -0.40978264808654785, 1.3768469095230103, 0.4154681861400604, -0.21451832354068756, -1.3141998052597046, -1.8192065954208374, -1.2331262826919556, -0.9029651284217834, -0.02246328629553318, 0.7673593759536743, 2.0685582160949707, -0.7229813933372498, 0.5204745531082153, 0.14627064764499664, 0.7359887957572937, -1.8282414674758911, 0.9559792876243591, -0.7711055874824524, -2.1878366470336914, -2.729656219482422, -0.4432661533355713, 1.2956963777542114, 1.1582237482070923, 1.1591695547103882, -0.5213496685028076, -0.07730372250080109, -1.4558993577957153, -0.8476482629776001, 0.07532288134098053, -1.1637402772903442, -0.5830276608467102, 1.0031802654266357, 0.3656845986843109, -2.2260613441467285, 2.733058214187622, -0.12468922883272171, -0.22637920081615448, 0.034620098769664764, 0.24612492322921753, -0.5724431872367859, 1.1966837644577026, -1.2087860107421875, 1.7260456085205078, 0.015324492007493973, -1.0359855890274048, 0.4978273808956146, -0.45495545864105225, -0.3251184821128845, 0.20422637462615967, -0.9605815410614014, -0.23801495134830475, 0.7361930012702942, 0.9235918521881104, -0.08106134831905365, 0.9635725617408752, -0.08858629316091537, 0.3364247679710388, 0.6260611414909363, -0.6515377163887024, -0.3068310022354126, 1.5049957036972046, 0.07203736156225204, 1.1703696250915527, -2.4374687671661377, 0.15115313231945038, -3.064161539077759, 0.735441267490387], [-1.2772542238235474, -0.6004554629325867, -0.36695441603660583, 0.5984697341918945, 0.05582484230399132, -0.28223446011543274, -0.3677375614643097, -0.9739097952842712, 1.0614680051803589, 0.12468903511762619, -1.6872345209121704, 0.5350081324577332, 0.30732694268226624, -1.3624498844146729, -0.8578709363937378, -0.0874275341629982, 0.23754385113716125, 0.6307183504104614, 1.9347710609436035, 1.201821208000183, 0.4630102813243866, 0.3157840073108673, -0.53549724817276, 0.21306362748146057, 0.8645868301391602, -0.48862653970718384, 0.22056233882904053, -1.4193756580352783, 0.26673126220703125, -0.4138535261154175, -1.9607824087142944, 0.5503795742988586, -0.5038363337516785, -0.15927591919898987, 1.3564485311508179, 0.3154430687427521, 0.8235997557640076, -0.026910122483968735, 1.0497040748596191, 0.02828400395810604, 0.3990265130996704, -1.5616871118545532, -0.559369683265686, -1.0516090393066406, 1.2819024324417114, -1.4305741786956787, 1.0584564208984375, -0.9128793478012085, -0.031055418774485588, -0.533944308757782, 0.2726038694381714, -0.012202592566609383, 1.9846957921981812, -0.6497330665588379, 1.652453899383545, -0.9756048917770386, -2.0465340614318848, 0.4115375876426697, -0.5543524622917175, 0.3551565408706665, -0.17701393365859985, 0.47590371966362, 2.440863609313965, -0.09596127271652222, 0.8180685639381409, 1.2623854875564575, -0.44468772411346436, -0.736576497554779, -0.9146571159362793, 1.274010181427002, -1.6902247667312622, -0.45080411434173584, 0.31931912899017334, 0.5179874897003174, 1.9282052516937256, 0.49476414918899536, 1.4451302289962769, 1.101428508758545, 0.1300099641084671, -1.8032996654510498, -1.3789747953414917, -0.4628024995326996, 0.26384928822517395, -1.1588383913040161, -0.4829900860786438, -0.6895598769187927, -0.3603284955024719, 0.6606153845787048, 1.5568751096725464, 0.2669486701488495, -0.6412111520767212, 1.3484513759613037, -1.166346549987793, -1.927065372467041, 0.10052265971899033, -1.0953705310821533, -0.025919470936059952, -1.0380626916885376, 1.6299285888671875, -0.7991737127304077, -0.16735993325710297, -0.289323091506958, 0.788162112236023, 0.5339588522911072, 0.7275087833404541, -0.5501753091812134, -0.0354209765791893, -0.5937364101409912, -0.7679827213287354, -0.05184876173734665, 0.8167505264282227, 1.3100008964538574, -0.7350267171859741, 0.4548266530036926, -1.0768120288848877, 0.39479678869247437, 3.177016019821167, 0.3565244674682617, -1.0154180526733398, -0.11328206956386566, 1.6101481914520264, 0.6960600018501282, -0.17418842017650604, -0.3323419392108917, -0.6248112916946411, -1.0966591835021973, -0.7596156597137451, 0.6788817048072815], [0.22069084644317627, 1.1227779388427734, -1.6121655702590942, -0.3935360312461853, -0.3192864954471588, -1.0435941219329834, 0.3536035418510437, -1.885079264640808, 0.23135590553283691, -0.7574580907821655, -1.2860864400863647, -1.4832215309143066, -0.425703763961792, 0.10481172800064087, -1.5425180196762085, 0.40630048513412476, 0.8484519720077515, -2.3221776485443115, 0.7891584634780884, 0.8618212342262268, 0.016684554517269135, -1.6716628074645996, -0.8995975255966187, -0.8703266382217407, 0.7448345422744751, 1.0797942876815796, 0.970641016960144, -0.019002137705683708, 0.9459446668624878, -0.7445005774497986, -0.48893043398857117, -1.9496561288833618, 0.5969160795211792, 0.24449950456619263, -0.13822691142559052, -1.508657693862915, 0.15063747763633728, -0.017298787832260132, -1.4906507730484009, 0.7018209099769592, -0.5477002263069153, 1.4481003284454346, 0.5733679533004761, -0.2931828796863556, -1.530636191368103, -0.03464854136109352, 0.12895749509334564, -0.4421267807483673, 0.47805166244506836, 0.29481396079063416, 1.2121667861938477, 1.7255016565322876, 0.42349979281425476, -0.08799952268600464, 1.9912476539611816, 1.5016429424285889, 0.2809780240058899, 1.7806520462036133, 0.28055840730667114, 0.8195304274559021, -0.5251573920249939, -0.7477943301200867, -1.0920790433883667, 0.5204623341560364, -0.7667753100395203, 1.11873459815979, 0.39170724153518677, 0.3617919683456421, -0.7464669346809387, -0.37995970249176025, -0.7460061311721802, 0.9257418513298035, 2.572763442993164, 1.3212945461273193, -1.2697083950042725, -0.3247450292110443, 0.6987540125846863, -2.156569480895996, -0.246599942445755, 0.9432148933410645, 0.17879116535186768, 0.1698647439479828, -0.18839110434055328, 1.318956971168518, 0.12768760323524475, 1.936170220375061, -0.9479113817214966, -0.08236677199602127, -0.7612722516059875, -1.3602821826934814, -0.22563916444778442, 0.837013840675354, -0.30898818373680115, 1.0516691207885742, 0.7305186986923218, -1.4849711656570435, 0.19273068010807037, -0.21516892313957214, 0.6502334475517273, 0.31733888387680054, 0.26853513717651367, -0.46700212359428406, -0.9167325496673584, 0.514087975025177, -0.5263895392417908, 1.2442289590835571, -1.4310753345489502, -0.6385411620140076, -1.316627025604248, 1.3307716846466064, -0.6570228338241577, 0.35307732224464417, -2.0653011798858643, 0.005085202399641275, -1.8570085763931274, 0.11327669769525528, 1.3836029767990112, -0.17217093706130981, -0.19602876901626587, -0.627919614315033, -0.23772013187408447, 1.391737937927246, -0.7689473032951355, -0.1851576864719391, 0.9268079400062561, 0.5712351202964783, -0.11004313826560974, -1.1243399381637573], [-1.0364384651184082, -0.4104169011116028, -0.2502953112125397, -1.2525019645690918, 0.7868131995201111, -0.9106033444404602, -1.6627798080444336, -0.9408959746360779, -0.41025620698928833, 1.5564699172973633, -0.14354869723320007, 1.7344354391098022, 0.22044281661510468, 0.07702808827161789, 0.4173220098018646, 1.243456244468689, 0.018552420660853386, 1.0940463542938232, 0.6110369563102722, -0.4590211808681488, -0.4806195795536041, 0.05722663924098015, -0.2003050148487091, 0.1362120807170868, 0.5338728427886963, 0.16714558005332947, -0.8258984088897705, 0.7084224820137024, -0.9077085256576538, 0.27506816387176514, -0.7828643321990967, 0.21011944115161896, 0.41686999797821045, 1.6841894388198853, 0.8031035661697388, -0.8584267497062683, 0.6205337643623352, 1.5250334739685059, 0.08142544329166412, -0.28708595037460327, 0.7503785490989685, -0.34183529019355774, -0.4023939073085785, 1.1549451351165771, -0.3468695878982544, -0.6274203658103943, 0.637144148349762, 0.33902305364608765, 0.3217643201351166, -2.0099313259124756, -0.6182420253753662, -0.7036944031715393, 2.1617939472198486, -0.744566798210144, -0.3905928134918213, 0.6532055139541626, -0.21870429813861847, -0.275589257478714, -0.5884777903556824, 1.4467129707336426, -0.6090418100357056, 0.26408979296684265, -0.47606393694877625, 0.9820674061775208, -2.323016405105591, 0.5469964146614075, -1.6595079898834229, -0.9213458299636841, 0.5379348993301392, -1.163611888885498, -0.9491575956344604, 1.070426106452942, -0.23007293045520782, -0.8908267021179199, -1.3441739082336426, -1.0845530033111572, 1.0677452087402344, -1.2194501161575317, -0.5874239802360535, -1.4473402500152588, 0.03279675915837288, 0.9949619174003601, -1.2992346286773682, 0.9153295755386353, 0.5782473087310791, -1.6649329662322998, -0.32523587346076965, -0.3052000105381012, 0.5633582472801208, -0.18011963367462158, -1.6164263486862183, 1.152004599571228, -0.8832975625991821, -0.14298103749752045, -0.04052416980266571, -1.6181846857070923, -0.39527592062950134, 1.8857859373092651, 0.36948034167289734, -0.3311631381511688, 0.7418729662895203, -1.1825870275497437, 0.01494542695581913, 1.5393414497375488, -1.9070345163345337, -2.0934951305389404, 0.6043210029602051, 2.7148211002349854, 0.18886195123195648, 0.501469612121582, -0.8042840361595154, -1.101264238357544, 1.4975895881652832, 2.9025862216949463, -1.0437554121017456, -0.009246909990906715, 0.8554442524909973, 1.2057844400405884, -1.3850017786026, -0.7516359090805054, -0.8996748924255371, 0.8700974583625793, 0.7331832647323608, -0.3053258955478668, 1.5738461017608643, -1.0540634393692017, -0.36914294958114624, -0.1977596879005432], [1.7093896865844727, 0.17762641608715057, -1.6638892889022827, -0.8283787369728088, -0.47937142848968506, -0.21089650690555573, 1.3852745294570923, -1.265601634979248, 0.1664281040430069, -0.8337776064872742, -0.2512521743774414, -0.5761017799377441, 0.4531947374343872, -1.0234578847885132, -0.48684003949165344, -1.1745151281356812, -0.33418744802474976, -1.516372561454773, 0.37195026874542236, 0.1303664743900299, 0.2614078223705292, 1.113566279411316, -0.7285796999931335, 1.3026307821273804, -0.8155444264411926, -2.155064344406128, -1.0418412685394287, 0.2614802420139313, 1.123289942741394, -1.4560520648956299, -0.304948627948761, -0.2413598895072937, 0.215408056974411, -1.4735925197601318, -0.5530213713645935, 0.19002172350883484, 0.8820708990097046, -2.4265451431274414, 1.1244481801986694, 0.013409068807959557, -0.5529003739356995, -1.2714673280715942, -0.08137746900320053, 0.4349002242088318, 0.4390483498573303, -0.9488611221313477, 0.40390753746032715, -0.00458852294832468, -0.15829181671142578, 0.9980976581573486, 2.3562066555023193, 1.1083202362060547, -0.9678314924240112, 0.5365726351737976, -0.7410812973976135, -2.758152484893799, -0.16956007480621338, -0.6030994653701782, 0.8332604169845581, -0.8749139904975891, 0.8898388147354126, -1.2574504613876343, -1.5992035865783691, 1.4489554166793823, -0.05080693960189819, 0.12555699050426483, -0.2563224732875824, 0.3294675052165985, -0.8223864436149597, 0.9377129673957825, 1.1973650455474854, 0.40041646361351013, -0.5468509793281555, -0.17072191834449768, 0.7518814206123352, 1.1911605596542358, -0.5659798979759216, 0.989707887172699, 0.08098214864730835, -1.044023871421814, 0.7071458697319031, 1.543013095855713, 0.07775463908910751, 0.6181771755218506, -0.06738745421171188, -1.1722378730773926, 1.0637195110321045, 0.72988361120224, -0.7988496422767639, 1.8538583517074585, 1.2973154783248901, -1.192906141281128, -1.0519237518310547, -0.38691672682762146, 1.5810388326644897, -0.46422386169433594, -0.920612633228302, -1.8047319650650024, 0.21278025209903717, -0.9305791854858398, 0.7512266635894775, 1.8207048177719116, -0.17740489542484283, -1.4606484174728394, 0.09372828155755997, 1.4233882427215576, -0.32528167963027954, 0.29362815618515015, -0.6366463899612427, 0.6849784255027771, -0.09752984344959259, 0.12275783717632294, -0.6267498135566711, -0.42484188079833984, -0.5707153081893921, -0.2911953032016754, -0.40922072529792786, 0.1006070151925087, 1.4517048597335815, 1.0799505710601807, 0.31824901700019836, -0.8808591961860657, -0.19514669477939606, -0.09308173507452011, 2.277710437774658, -2.0722949504852295, 0.16462166607379913, -1.3922888040542603], [-0.8403884768486023, 0.11809840053319931, -0.04731703922152519, 2.348539113998413, 0.872705340385437, -0.7111290693283081, -0.012487152591347694, -0.7673301696777344, 1.362040400505066, -0.25180143117904663, 1.4333418607711792, 1.4669129848480225, -3.6658859252929688, -0.2035481333732605, 0.6380649209022522, -1.2287863492965698, 0.4370046555995941, -0.24010345339775085, 0.20064912736415863, -0.5471188426017761, 0.25642406940460205, 1.4672666788101196, -0.1871812492609024, 1.3387645483016968, -0.10183914750814438, -1.7676236629486084, -0.7190889716148376, -0.8408563137054443, -1.3077040910720825, -0.7071720957756042, 2.131293296813965, -0.3644091486930847, 1.9077085256576538, -0.37044090032577515, -0.48224350810050964, -2.138446569442749, 0.3833088278770447, 0.43024569749832153, 0.3018610179424286, 0.14251647889614105, -1.3877071142196655, 1.2413405179977417, 1.4947649240493774, 0.015230047516524792, -0.5466383695602417, -0.5231532454490662, 0.9959595203399658, -0.4877004325389862, -1.1042022705078125, -0.6920278072357178, -0.49214890599250793, 0.689348578453064, 1.6296308040618896, -0.18596772849559784, 0.8996768593788147, 0.128932923078537, -1.0370848178863525, 0.7675752639770508, 0.22859302163124084, -1.0723199844360352, -0.47302308678627014, -0.45838242769241333, 0.1871953159570694, -0.1424056738615036, -0.4285300374031067, -1.4432579278945923, -0.17855045199394226, 0.3431466817855835, -1.3141376972198486, 0.4477895498275757, 1.1186096668243408, -0.2796369194984436, -0.5524203181266785, 0.15789368748664856, 0.9040833115577698, -0.22767993807792664, 0.6905876994132996, 1.7594534158706665, 1.024214267730713, 0.8220387101173401, 0.004419934935867786, -0.9582284688949585, -0.4096059203147888, 0.1517903059720993, -0.2736327052116394, -0.1008412167429924, -0.6374163031578064, 0.6336201429367065, 0.12824469804763794, -0.44684094190597534, 0.1656630039215088, 1.1246346235275269, 2.0578043460845947, 1.1539604663848877, 1.2887537479400635, 0.9436708688735962, 1.6802520751953125, -0.7303187847137451, 1.6889415979385376, 0.9114269018173218, 1.0173879861831665, 0.536426305770874, -2.073625087738037, -0.029545791447162628, -0.2903314530849457, -0.36696290969848633, -1.2015377283096313, -0.7192136645317078, 1.3313148021697998, -1.0829013586044312, 0.5756344795227051, -0.7792432904243469, -2.4212207794189453, 0.8495041728019714, 1.3408437967300415, -1.6059037446975708, 1.188924789428711, -1.5669975280761719, -1.046701431274414, -0.7396310567855835, 0.8482450842857361, -0.018274866044521332, 0.05683941766619682, 0.673904538154602, -0.30628958344459534, 0.44765785336494446, -0.847792387008667, 1.3397740125656128]]]], "dtype": "float32"}, {"__nd__": [[[[0.8736039996147156, 0.15731936693191528, 0.6875300407409668, -0.12005700170993805, -0.7395879626274109, -0.2597513198852539, -0.7716795206069946, -1.2318142652511597, 0.2216801792383194, 0.013803155161440372, -1.9095252752304077, -0.7770386338233948, -0.685000479221344, 0.006419667508453131, 1.675560474395752, -0.5384746193885803, -0.6377695202827454, -0.6754165887832642, 0.5895214676856995, 1.046329140663147, -0.31823837757110596, -0.6339491009712219, 0.4911782145500183, -0.37347137928009033, 0.8282777070999146, -1.703218698501587, -0.31221815943717957, -0.0243344996124506, 0.07835352420806885, -0.6220927238464355, 0.31472596526145935, 0.9638266563415527, -0.27920153737068176, -0.7138554453849792, 0.34674766659736633, 0.7846325039863586, -0.0991014614701271, -1.8570845127105713, 1.2253729104995728, -0.8912966251373291, -0.271480530500412, -2.438021659851074, 0.6858837604522705, 0.8134121894836426, -0.20889516174793243, -0.7200455665588379, 1.2029260396957397, -0.25584572553634644, -0.9550201892852783, -0.21053285896778107, 0.5546116232872009, -1.0595520734786987, 0.6627993583679199, -1.1942813396453857, -1.488598108291626, 0.5653240084648132, 1.1102819442749023, -1.3289159536361694, -1.9398627281188965, 1.8079006671905518, -0.5966861844062805, -1.1470540761947632, 0.7123070359230042, 0.025793297216296196, 0.16077691316604614, -0.3601173460483551, -1.2457104921340942, -1.1827094554901123, 0.9613455533981323, 0.7189039587974548, 0.9330087304115295, 0.05873674526810646, 0.6094839572906494, 0.24679623544216156, -0.3582583963871002, 0.7225037217140198, -0.47776153683662415, 0.027546389028429985, -2.1546576023101807, -0.045462850481271744, 0.07580691576004028, 0.061164166778326035, -1.4012675285339355, -0.43210652470588684, -1.495564341545105, -0.5231676697731018, -0.44172191619873047, -0.5283340811729431, -0.9000887870788574, -0.20933528244495392, 1.6438946723937988, 0.18106037378311157, -1.0310934782028198, 1.669778823852539, 0.5008178949356079, 0.028434157371520996, -2.245677947998047, 1.3082104921340942, -1.3940831422805786, 0.5710281133651733, -1.5829732418060303, 0.9836093187332153, -0.1659993976354599, -0.23785588145256042, 1.2592928409576416, -1.0423541069030762, -1.0450700521469116, -0.7531119585037231, 0.197585791349411, -0.8500763177871704, 0.369705468416214, 1.1607835292816162, -2.2201802730560303, 0.8130172491073608, -0.2151542752981186, -0.5665802359580994, -0.19740906357765198, 0.5671038627624512, -0.3805658519268036, -0.5157858729362488, -0.592960000038147, -0.39040493965148926, 0.2767347991466522, 0.9752312302589417, -1.9994666576385498, -0.6698939800262451, 1.1935036182403564, -0.4925214350223541], [0.6824780106544495, 0.4165710508823395, 1.1816260814666748, 0.6546476483345032, 0.36207905411720276, -1.9078847169876099, 1.231790542602539, 0.3391273319721222, 1.0999832153320312, 1.3020752668380737, -0.2807355225086212, -0.2959780693054199, 0.4575335383415222, -1.1958822011947632, 0.7165471911430359, -1.325432300567627, -0.18396946787834167, 0.20252689719200134, -0.11657962203025818, -1.400212049484253, -1.506413221359253, 2.0935769081115723, 1.1816813945770264, 0.5552921891212463, -0.22532057762145996, -0.5459766983985901, 0.9640997648239136, -0.45759648084640503, 0.4041130542755127, 0.8062846064567566, -0.661961555480957, -1.8096684217453003, 1.387045979499817, -0.42715445160865784, -0.2820322513580322, -0.8348677158355713, -1.684698462486267, -1.0280345678329468, 1.0259698629379272, 0.9422712922096252, -0.9794763922691345, 0.26586389541625977, 0.8447336554527283, 1.0468199253082275, -0.23385922610759735, -0.7762040495872498, 1.8548836708068848, -0.001610904699191451, 1.3657667636871338, -0.026136677712202072, 1.6601659059524536, -1.1525063514709473, 0.110784150660038, -1.6175427436828613, -0.8435966968536377, -0.35787278413772583, -0.6078000664710999, -0.11971721053123474, -0.14436408877372742, 1.8080672025680542, -0.768817663192749, -0.2437857985496521, 0.8917720317840576, 1.3117607831954956, -1.056397557258606, -0.21471938490867615, -1.2878007888793945, -1.1478900909423828, -0.09158521890640259, 0.8807429075241089, 0.10759372264146805, -0.05185862258076668, -0.16310882568359375, -1.4188826084136963, 0.33735010027885437, -1.1040197610855103, 0.9037536978721619, 0.18449223041534424, -1.246993064880371, -1.0158414840698242, -0.4140991270542145, -1.4899924993515015, 0.41686466336250305, 2.4727425575256348, 0.39501702785491943, -0.25549671053886414, 0.6170067191123962, -1.473385214805603, 2.1733200550079346, 0.6982879638671875, -1.0684787034988403, 0.6656769514083862, 1.519181728363037, -1.0787323713302612, 0.473641961812973, 0.18703581392765045, 0.3161229193210602, 1.7369294166564941, -0.8216834664344788, 1.36632239818573, -0.5171881318092346, -1.250537395477295, 0.35668501257896423, -0.27632081508636475, 0.029226716607809067, -0.9090926647186279, 0.6907795667648315, -0.6223958730697632, -0.54206782579422, -0.22325918078422546, 0.015320472419261932, 0.6374498009681702, 1.0116239786148071, -0.3230643570423126, 0.356499582529068, 1.3146612644195557, -0.8432985544204712, -2.249885320663452, -0.19804924726486206, -1.4359498023986816, -2.1212217807769775, -1.0685194730758667, 2.50421142578125, 0.7013607025146484, -0.66145920753479, -0.7609538435935974, -1.1069684028625488, -0.7940318584442139], [0.4065946638584137, 0.6171542406082153, -2.0283970832824707, 1.2506335973739624, 1.7638858556747437, -1.2518430948257446, -1.1219887733459473, 0.4680090844631195, 1.7783620357513428, 1.5831966400146484, 1.0804067850112915, -0.2589225769042969, -0.5981753468513489, -0.40217894315719604, -0.9467190504074097, -1.6045902967453003, 1.9373294115066528, -0.05943283066153526, 1.2227312326431274, 0.8973038196563721, -0.2971752882003784, 1.2484252452850342, 0.10807035863399506, -0.30515173077583313, 0.8942939639091492, 0.5314021110534668, 1.6027653217315674, -0.7887691855430603, -0.7424569725990295, 0.3558101952075958, 0.8879269957542419, -0.4861353933811188, -2.497466802597046, 0.3180462121963501, 2.0955214500427246, -0.5350421667098999, -1.6323319673538208, -1.0285365581512451, -0.1447240114212036, 0.5275033116340637, -0.6099622249603271, 0.3049418330192566, 0.33063483238220215, -0.9276916980743408, 0.8687241673469543, 1.2362498044967651, -0.976148784160614, 1.3297172784805298, -1.3014744520187378, 0.4163948893547058, -0.29895880818367004, 0.24560396373271942, -1.192744493484497, 0.8815432786941528, 0.5481248497962952, -1.0198428630828857, -0.6291729807853699, -0.8276495933532715, -0.786413311958313, 0.5169771313667297, 0.5206230878829956, 0.47153905034065247, -0.4855846166610718, 0.24723246693611145, -1.6289509534835815, -1.1830220222473145, -1.8581737279891968, 0.5272151827812195, 1.3013100624084473, -0.7867207527160645, 1.5480024814605713, -0.052183136343955994, 1.2262283563613892, 0.37691041827201843, 0.07312940061092377, 0.7732512354850769, 0.8184435367584229, -0.8797794580459595, -0.5762750506401062, -0.14082977175712585, 1.574554204940796, 0.1578560471534729, -1.1880805492401123, 0.6983698010444641, -1.207075834274292, 0.08944835513830185, 0.23038384318351746, 0.41412168741226196, -1.3170990943908691, -1.1816197633743286, 1.1277174949645996, 0.8636512756347656, -0.11212629824876785, -0.5953447818756104, -0.3979322016239166, 2.719053030014038, 1.186305046081543, 0.5459256172180176, -0.9659627676010132, -0.4254215359687805, -0.10740496963262558, -1.8220970630645752, -1.3225330114364624, 0.04490720480680466, -0.592380702495575, 1.5770182609558105, 1.1816432476043701, 0.08156898617744446, 1.4318358898162842, -0.7042598128318787, -0.9157237410545349, -0.6987130641937256, 1.3431487083435059, 0.353674978017807, 1.0383840799331665, -0.35599350929260254, -1.1999534368515015, 0.060645051300525665, -0.18419009447097778, -0.27348268032073975, 1.8165754079818726, -0.6501209139823914, 0.6875178217887878, -0.33564865589141846, -0.7221391797065735, 1.5697945356369019, -0.1699417382478714, 1.474066138267517], [1.3012970685958862, -0.8890240788459778, 0.6835005283355713, 2.8303020000457764, -0.6483827829360962, 0.16606703400611877, 0.39159050583839417, -0.8755418062210083, 0.5349100232124329, 0.14608368277549744, 0.06019379571080208, 0.7796676754951477, 0.13819414377212524, 0.7665284872055054, 0.8514993786811829, 0.17680881917476654, -0.04323658347129822, -1.5048311948776245, -0.9459723234176636, -1.5145063400268555, 0.7848822474479675, -0.5973302125930786, 0.10567965358495712, 0.047784700989723206, -1.1699966192245483, -0.721991240978241, 1.8100742101669312, 0.24766388535499573, 0.23004776239395142, -0.041767507791519165, -0.3272996246814728, 0.5786759853363037, 0.7877863049507141, 0.9294033050537109, 0.10463184863328934, 0.15765038132667542, 1.380251407623291, -0.9888811111450195, 1.0590108633041382, -0.08243648707866669, -1.259773850440979, -1.6241698265075684, 0.08294868469238281, -0.13501860201358795, 0.35312697291374207, -0.3006771504878998, 0.5138839483261108, 1.9855618476867676, 1.3350355625152588, 0.8739362955093384, -0.001192632014863193, 1.2000539302825928, -0.23995789885520935, -0.5990977883338928, -0.29216110706329346, 0.09660319983959198, -0.38024815917015076, -0.9910969734191895, 1.3740732669830322, 2.4338316917419434, 1.6602383852005005, 0.1522144228219986, -0.5168999433517456, -0.10881847143173218, 0.8602548837661743, 0.8478158116340637, 0.39974066615104675, -1.1496282815933228, 0.0639321431517601, 0.825592577457428, -1.601277470588684, 0.965656578540802, -0.26769784092903137, 1.0633667707443237, -0.5730745196342468, 1.0231432914733887, -0.25304552912712097, -0.6254022121429443, 0.6025553345680237, -0.31015297770500183, -0.6998600959777832, -1.3629353046417236, -0.3074309527873993, 1.775692105293274, 0.05505833774805069, -0.39591267704963684, -0.28442543745040894, -0.7466901540756226, 0.19005727767944336, -2.06314754486084, -0.1077672466635704, -0.9096394181251526, -0.3229798376560211, -0.6932346820831299, 1.211177945137024, -0.34867021441459656, 0.14315539598464966, 0.43431946635246277, 1.3381205797195435, -1.9311847686767578, 0.25455403327941895, -1.7230123281478882, -1.313188910484314, -0.8204591274261475, 0.07307598739862442, -1.1831036806106567, -0.30085548758506775, -1.609971046447754, 0.9919850826263428, -0.11219299584627151, 0.3302004933357239, -0.5985848307609558, 0.7907562255859375, -0.7245619297027588, 0.8571275472640991, 1.2583467960357666, -0.4735427796840668, 0.1590729057788849, 0.5442526936531067, -0.16274864971637726, 0.9265170693397522, -0.6689677238464355, 0.034813184291124344, -0.7040562629699707, 0.36766940355300903, -0.0681406557559967, 0.8028731942176819, 0.19728584587574005], [0.5166002511978149, 0.3718350827693939, 0.5176395177841187, 0.11119705438613892, -0.004749227315187454, -1.1676005125045776, 1.360603928565979, -0.4038056433200836, 0.572826623916626, 0.2530311644077301, -0.5654471516609192, -1.499622106552124, 0.3562358617782593, 0.331592321395874, 0.2373192459344864, 0.44809624552726746, -0.8404109477996826, -0.36196282505989075, 0.27285969257354736, -0.11520521342754364, -0.9187480211257935, 0.36821404099464417, -0.8519102334976196, -1.3671026229858398, -0.516308605670929, -1.0433123111724854, 1.2426152229309082, 2.357131004333496, -0.36979326605796814, 0.04558803141117096, -0.03646297752857208, -0.9951309561729431, 0.390930712223053, -0.8019903302192688, 0.886015772819519, -1.1429980993270874, -0.6653619408607483, -0.954255223274231, 0.1768747717142105, -0.18340061604976654, 1.0326870679855347, 1.0460797548294067, 0.5859891772270203, -0.8649829030036926, -0.27247440814971924, -0.959337055683136, -0.760452151298523, 0.07885921746492386, -0.03504712134599686, 0.20234984159469604, 1.3854775428771973, -0.12494593858718872, -0.08066000789403915, -0.27412497997283936, 1.1225221157073975, 1.0569472312927246, 0.7079856395721436, -0.8468422293663025, 1.0863869190216064, 0.8320211172103882, 0.8153210282325745, 0.7014232277870178, 1.514814853668213, -2.2363998889923096, 2.1609065532684326, -0.3078746795654297, -0.9821572303771973, -0.11018890142440796, -0.06890785694122314, -0.24972566962242126, 1.6848200559616089, -2.66766357421875, -0.34749647974967957, 1.5996328592300415, -0.7952759265899658, 1.4165592193603516, -2.3512473106384277, 0.8920450210571289, 1.5482233762741089, -1.8875213861465454, 0.9216681718826294, 2.792330741882324, 0.4758397042751312, 0.6912760734558105, -0.40839454531669617, -1.044775128364563, 0.5751681923866272, -1.334529995918274, 0.23123255372047424, -0.41078871488571167, 0.40440690517425537, -0.0017909164307639003, -0.8906059861183167, -0.1045050173997879, -0.6978714466094971, 1.0154128074645996, 0.1632758229970932, -0.43405014276504517, -0.8757899403572083, 0.3432176113128662, 0.15528512001037598, 1.3319987058639526, -1.1202492713928223, 0.16405019164085388, -0.8400053381919861, -0.9361001253128052, -1.3599554300308228, 0.4941588044166565, -2.135838747024536, -0.02947058156132698, 0.24062038958072662, 1.4476206302642822, -0.2286737859249115, 0.45134204626083374, -0.28658396005630493, -1.2185307741165161, 0.4968215823173523, -0.33734193444252014, 0.17840702831745148, 1.8156355619430542, 1.492779016494751, 0.2617633640766144, 2.1378252506256104, -0.16100306808948517, 0.5877252221107483, -0.423808217048645, -1.4786405563354492, 1.6108839511871338], [-1.117783546447754, 0.2583378255367279, -0.8620555996894836, -0.27807870507240295, -1.3252445459365845, -0.9547833204269409, -1.4190881252288818, 0.6317204833030701, -0.3686351180076599, 1.1173620223999023, 1.3890163898468018, -0.11721374094486237, -0.045996129512786865, 0.7933138608932495, 1.5849779844284058, -0.22262690961360931, -0.757912814617157, 0.372830331325531, 0.8992161750793457, 1.638999342918396, 1.8722319602966309, 0.9694207906723022, 0.23210687935352325, -1.9624967575073242, 0.7664762139320374, 1.572528600692749, 0.8893353939056396, 0.9592294692993164, 1.6999993324279785, -1.5291961431503296, 0.19693216681480408, 0.20241282880306244, 0.9264315962791443, -0.8086308240890503, -0.21088099479675293, 0.4463173747062683, 0.31820183992385864, 0.8507063388824463, 0.7859904170036316, 0.5344428420066833, 0.138225719332695, 1.4636574983596802, 0.03868463262915611, -1.0083527565002441, 0.17603641748428345, -1.8352148532867432, 1.168584942817688, -0.08611323684453964, -1.7018952369689941, -0.698564887046814, -0.12053867429494858, -0.9965546131134033, 0.5010359883308411, 0.8550175428390503, 0.49521133303642273, -1.192097544670105, -0.09845723956823349, -0.4542355239391327, 0.49544528126716614, -0.9874755144119263, -1.1602580547332764, -0.34901565313339233, 0.24515272676944733, 1.3953659534454346, -0.3494727611541748, 0.09475841373205185, 0.6315497756004333, -1.4747263193130493, 0.7554972171783447, 1.1172930002212524, 0.5183646082878113, 0.5822424292564392, 1.1218338012695312, -0.15130199491977692, 1.2866098880767822, -0.23108887672424316, -0.8162621259689331, 1.4057267904281616, -1.278069019317627, 0.6845436692237854, 0.9640765190124512, -0.2875904142856598, -1.755411148071289, 0.2856196463108063, -0.5476337671279907, -0.016195565462112427, 1.2140156030654907, -0.2566925287246704, -1.452238917350769, -0.16855059564113617, -1.24703848361969, 1.4792020320892334, -0.710994303226471, 0.11654072254896164, -1.6311116218566895, -0.9516277313232422, -1.1284867525100708, 1.0764964818954468, -0.4516611695289612, 0.9986801743507385, 0.16090494394302368, -0.20808695256710052, 0.02564949169754982, 0.33557164669036865, -0.8825423717498779, 0.9829254150390625, 0.26650723814964294, 1.768456220626831, 0.940321683883667, 0.2375335842370987, -0.6501938700675964, -1.1208994388580322, 0.08989863097667694, -0.6162654757499695, 0.2726679742336273, -2.6115012168884277, 0.2509326636791229, 0.444291889667511, 0.7216050028800964, -0.6555115580558777, 2.4139537811279297, -2.5495352745056152, 0.0038522661197930574, 0.7423458695411682, -1.1957489252090454, -1.378790259361267, -0.1661055088043213, 2.0808286666870117], [0.26975324749946594, -1.3782140016555786, -1.5780853033065796, 0.7537512183189392, 0.7562263607978821, -0.6172135472297668, -0.033351123332977295, -0.26854997873306274, -0.6629060506820679, -1.6256717443466187, 1.2223899364471436, -0.8816549181938171, 0.5831782221794128, -0.16053487360477448, 0.24458765983581543, 0.1826721876859665, 0.9590024352073669, 0.90067058801651, 0.6570497155189514, -0.2897438406944275, 0.15662138164043427, -1.832996129989624, -1.632547378540039, -1.5367472171783447, -0.8893078565597534, -1.1430630683898926, 0.7563406825065613, 0.5090059638023376, 0.5813795924186707, -1.1990818977355957, -0.04636207967996597, 0.8851603865623474, 0.32352468371391296, -1.4427577257156372, -0.694783627986908, 0.24931250512599945, 1.2002547979354858, -2.2451486587524414, 0.26354894042015076, -0.23323078453540802, -0.037240512669086456, -0.586685836315155, 0.8964551091194153, -0.23936833441257477, 0.41726604104042053, 0.6686733961105347, 0.11193254590034485, -0.14293590188026428, -0.005418695509433746, -1.3060581684112549, 0.8595054149627686, -0.7195802330970764, 0.39530351758003235, -0.27256298065185547, -0.01293297577649355, 0.5490997433662415, -1.626565933227539, 0.3123602569103241, -1.2774578332901, -0.7746238112449646, -0.42035970091819763, 1.1406402587890625, 1.1911680698394775, 0.5326961278915405, -0.9925504326820374, 1.1240569353103638, -0.9870653748512268, 0.14921872317790985, -1.8869688510894775, -1.5446977615356445, 1.730586051940918, -0.9421936273574829, -1.3568581342697144, -1.5776346921920776, -0.222887322306633, -0.17376556992530823, -0.451029509305954, -0.35893186926841736, 0.7147130966186523, -0.5435191988945007, -0.46360278129577637, 0.16243283450603485, -0.3404199779033661, -1.0434914827346802, 0.11778455972671509, -0.09511963278055191, 0.5507141947746277, 0.171688050031662, 1.0110729932785034, 1.6189844608306885, 0.1780090183019638, -0.8539320230484009, -0.16745525598526, 0.540292501449585, 1.6408629417419434, 0.7945625185966492, -0.3684236705303192, -0.20177501440048218, -0.029489992186427116, 0.4293792247772217, 0.1454893946647644, 0.8756510019302368, -0.5479726195335388, 2.000422716140747, -0.3868756592273712, -0.24270936846733093, -0.2933714687824249, -1.0673233270645142, -0.35090330243110657, -1.387742042541504, 0.27679842710494995, -1.049849033355713, -0.3472283184528351, 0.6008750200271606, -0.9468904137611389, -1.8701155185699463, -0.08332003653049469, 0.8555158376693726, 0.6439535617828369, 0.5975571274757385, 0.6374019384384155, 1.4783337116241455, 0.29593825340270996, 0.778015673160553, 0.34663692116737366, -3.0357954502105713, -0.8134716749191284, -1.3131182193756104], [-0.781239926815033, -0.4480955898761749, 0.8756301999092102, 0.06759214401245117, -0.00833248533308506, 0.13325391709804535, 0.05149165168404579, -0.706436276435852, -0.5862107276916504, -0.7072077393531799, -0.7627639770507812, 1.1757054328918457, -1.5674501657485962, -1.7549195289611816, -1.343733787536621, -1.492558240890503, 0.9218448996543884, -0.5166969895362854, 2.099148988723755, -0.26364365220069885, -0.41201475262641907, -0.5965595245361328, -0.06204599142074585, 1.115903615951538, -2.269744396209717, 0.720170795917511, 1.0457727909088135, -1.0958666801452637, 0.523259162902832, 0.06416439265012741, -0.6917681097984314, 0.2614745795726776, 0.5986984372138977, 1.2060130834579468, -1.4465311765670776, -0.21665380895137787, 0.9392386674880981, -0.9418367147445679, 0.7422623038291931, 0.6994324922561646, -0.9482300877571106, 1.042739987373352, -3.457988977432251, 0.691627025604248, -0.8423312902450562, -0.4852445721626282, -0.28437191247940063, 0.33608880639076233, -0.6852841377258301, 0.6197010278701782, 0.8272746801376343, -0.03927009552717209, -0.2814134955406189, -1.9382643699645996, -0.029087290167808533, -0.9384210109710693, -0.8756120204925537, -1.6371015310287476, -0.5905622243881226, 0.027955187484622, -0.4701286256313324, 1.794097661972046, 1.628124475479126, 0.08666250109672546, -0.5102596879005432, -1.4698251485824585, 0.011968446895480156, 0.21195025742053986, -1.343667984008789, 0.4760398864746094, 0.3433624505996704, 0.4779590964317322, 0.3673486113548279, -1.9347695112228394, 1.7444374561309814, 1.1323024034500122, -1.037310242652893, -0.43029963970184326, -0.31810665130615234, 1.0374356508255005, -0.8930041193962097, -1.0436605215072632, -0.3211021423339844, 0.6518227458000183, 0.9257311820983887, -0.4940741956233978, 1.4687016010284424, 0.16704526543617249, 0.02523861639201641, 0.7431730628013611, 0.2098592072725296, -0.691521942615509, -0.7875882387161255, 0.022149881348013878, -0.24717780947685242, 1.6555497646331787, 1.8341660499572754, 0.06086762621998787, -1.9766634702682495, -1.4026517868041992, 0.1464761644601822, 0.053265318274497986, -1.9221206903457642, 0.038296692073345184, 0.20236243307590485, -1.7692099809646606, -1.5850701332092285, -1.1805850267410278, 0.09069585055112839, -0.3204841613769531, 0.4247019290924072, 0.8366325497627258, 0.11556188017129898, -1.1972641944885254, -0.9985888600349426, 1.8526031970977783, 1.3200496435165405, -0.5346185564994812, -1.5409576892852783, 0.41377368569374084, 1.2193537950515747, 0.3464807868003845, -0.36850082874298096, -0.37950488924980164, 2.0961344242095947, -0.6981217861175537, -1.3669148683547974, 0.3451535403728485], [-0.0426984541118145, 0.4695527255535126, 0.49639254808425903, -0.6126411557197571, 0.897931694984436, -1.0556124448776245, -0.5808061957359314, -1.2073365449905396, -1.439803957939148, 0.7000480890274048, -1.4850678443908691, 1.1889311075210571, 0.46832799911499023, 0.34551501274108887, -1.112715244293213, -0.2242773324251175, -0.17864039540290833, 0.3783039450645447, 1.0381596088409424, -2.1573946475982666, -0.9990082383155823, 0.25373077392578125, 0.3917118012905121, -1.0313564538955688, 0.666218101978302, -0.08480941504240036, -0.22236758470535278, 0.363966166973114, -0.5612441897392273, -0.6057363748550415, -1.375095009803772, 1.5691742897033691, 0.3159302771091461, -1.2808934450149536, -0.9720355868339539, 2.0185344219207764, 0.33609044551849365, 0.2614986300468445, -0.8470391631126404, 0.6867311596870422, 0.7709226012229919, -1.1867233514785767, -0.3950383961200714, -0.7017769813537598, 0.7625503540039062, -0.11483205109834671, -0.19437631964683533, 0.13993556797504425, 0.6031279563903809, -1.0499061346054077, 1.5378801822662354, -0.16907289624214172, -0.14758175611495972, -1.458484172821045, -1.6633930206298828, 0.20768921077251434, 0.2661805748939514, 0.47044631838798523, -2.2246921062469482, 0.24490006268024445, -0.9594683647155762, 0.7904053926467896, -1.8115897178649902, 0.6413095593452454, 0.19011452794075012, 0.47639116644859314, 0.9507083892822266, 0.11843665689229965, 1.6924101114273071, -1.22482430934906, -0.29754212498664856, 0.4483836591243744, 0.49716928601264954, -0.8474916219711304, -1.0676524639129639, -0.38986754417419434, -0.19173140823841095, 1.4645262956619263, 0.5189773440361023, 1.3859483003616333, -0.7158039212226868, -0.4919881522655487, 0.26580071449279785, -0.418558269739151, 0.22932758927345276, -0.7989467978477478, -0.2742254436016083, 0.6920485496520996, -1.943447232246399, -0.9240918159484863, 0.2922920286655426, -0.6965076327323914, -0.7507445812225342, -0.5524939298629761, 0.06272586435079575, 1.075475811958313, 1.418715000152588, -0.16410017013549805, 0.45131587982177734, 1.46134352684021, 1.3139979839324951, -0.8449932336807251, 1.5204799175262451, -1.5551916360855103, 0.8719222545623779, 1.5940454006195068, -0.8540745377540588, 0.5090672373771667, 1.065971851348877, 1.3992787599563599, 0.394603967666626, 0.34209775924682617, 0.550442099571228, 0.8595373630523682, -1.193722128868103, -0.8405243158340454, 0.7707321643829346, 2.4078657627105713, 0.5412307977676392, -1.2819668054580688, -0.24646662175655365, -0.3893234133720398, 1.9807778596878052, 0.25440651178359985, -0.12503454089164734, -1.7238187789916992, -0.32837674021720886, 0.9219086170196533], [-0.17425812780857086, -0.48879584670066833, -0.38473114371299744, -0.9204152822494507, 1.9380922317504883, -0.31533291935920715, 1.3701194524765015, -0.1368340402841568, 1.1142245531082153, 1.6463171243667603, -0.7282148003578186, -0.29442456364631653, -0.3425503671169281, -1.895259141921997, -0.04464724287390709, 1.0898911952972412, -0.2584564685821533, -0.7773514986038208, -0.5583814382553101, -1.2044278383255005, -0.025121888145804405, -0.2621970772743225, 0.34923291206359863, -0.1218113973736763, -0.04031406342983246, 0.5251116156578064, -1.2372663021087646, 0.30130165815353394, 1.685643196105957, 0.10138348489999771, 1.3455591201782227, -0.9954193234443665, 1.147020697593689, -1.1254620552062988, -0.9495582580566406, -1.1480058431625366, 0.5254939198493958, -1.1841274499893188, 0.18573445081710815, 2.2239367961883545, -0.2841190695762634, -1.6474288702011108, -0.10725840926170349, -0.24807575345039368, -0.5507386326789856, 0.7601838707923889, 0.8413646817207336, 0.3068012595176697, 0.337480366230011, -1.0864869356155396, 0.18785999715328217, 0.7041370868682861, 0.5728713274002075, -0.26898130774497986, 0.5395395159721375, 1.5256600379943848, -0.14158281683921814, -0.0302211232483387, -0.3942800760269165, -0.6112408638000488, 1.122680425643921, 0.7794691324234009, 0.10523494333028793, 1.5492743253707886, 0.6690903902053833, 0.981644332408905, -1.616376280784607, -1.1565359830856323, -0.1571090817451477, 1.0150226354599, -0.3733052909374237, 0.4104527235031128, 0.1386667788028717, -0.14221897721290588, -1.5506104230880737, -1.4159198999404907, -0.20441210269927979, -0.15679748356342316, 0.12954339385032654, 0.1343768686056137, 0.036372844129800797, -0.4987471401691437, -0.15998950600624084, 0.3153054416179657, 0.5232757329940796, 1.227549433708191, 0.6472606062889099, 1.2085622549057007, 1.0747193098068237, -0.11423817276954651, 0.5188882350921631, 0.7129391431808472, -0.8860849738121033, 1.090954303741455, 0.346873015165329, 0.7392697930335999, 0.4809889495372772, 0.8232365250587463, 1.2400085926055908, 0.0848560631275177, -0.5378279089927673, -0.09370596706867218, 0.12947174906730652, 0.25335168838500977, 0.6176750063896179, -0.7887099385261536, 0.9570348262786865, -1.8273906707763672, -1.273972749710083, 0.9962722063064575, -0.6434611678123474, 1.6990920305252075, 0.0047663431614637375, -0.13096261024475098, -0.8248211741447449, -0.205110102891922, 0.3570122718811035, 0.26134157180786133, -0.2717079222202301, 1.3950282335281372, -0.19216513633728027, 0.632013738155365, -0.3375694453716278, 0.7108540534973145, -0.7930124402046204, 0.41373714804649353, -0.6139930486679077, -0.09225893765687943], [-0.05411228537559509, -2.243133544921875, -0.6891900300979614, -0.50473552942276, 0.3267310559749603, 0.03887812793254852, 0.44664523005485535, 1.2734378576278687, -1.9207935333251953, -0.7133700847625732, 0.6074567437171936, 0.01352581474930048, -0.739717423915863, 0.6406126618385315, -0.048774074763059616, 0.6497613787651062, 0.30543774366378784, -0.526313304901123, -1.2597719430923462, 0.6584674119949341, -1.5501432418823242, -0.346099853515625, -0.25892776250839233, 0.7292901277542114, 0.5480188131332397, 0.35936376452445984, -2.059908390045166, -1.9320746660232544, -0.057053904980421066, 1.4714869260787964, 2.369520902633667, 0.9167271256446838, 1.5548278093338013, -0.07900483161211014, 1.0586050748825073, 1.1214137077331543, 0.8418004512786865, -0.24046781659126282, 0.14641210436820984, 0.34646740555763245, 1.3252671957015991, 1.0744277238845825, 0.7978953123092651, 1.1806139945983887, -1.3746238946914673, -0.5997911095619202, -1.8605538606643677, -2.5534040927886963, 0.06378492712974548, -0.6331507563591003, -1.388122797012329, 0.196604385972023, 2.183788776397705, -0.8230065107345581, -0.017527567222714424, 1.5794743299484253, 0.46399614214897156, -0.5743453502655029, 0.36614012718200684, 0.2033238559961319, -0.1879357099533081, 0.1204429492354393, -1.4574836492538452, -0.2709541320800781, 0.42631304264068604, -0.6105365753173828, -0.64340740442276, -1.1480125188827515, 0.017676569521427155, -0.011315904557704926, -0.42798173427581787, 0.795068085193634, 1.5721662044525146, 0.08293025195598602, -0.42060530185699463, 0.2651497423648834, -0.8181062936782837, -1.302964448928833, -0.6515328288078308, 0.47713392972946167, -0.6002343893051147, 0.20606230199337006, 1.4941151142120361, 1.6384271383285522, 0.3874424695968628, -1.162537693977356, 0.1676367074251175, -1.107696533203125, -0.7519698739051819, -1.452393889427185, -1.6612497568130493, -0.24492159485816956, -0.35450276732444763, 0.2394016832113266, -0.29722660779953003, 0.9024902582168579, -1.3278748989105225, -0.24200554192066193, 0.195702463388443, 0.36913391947746277, 0.5723692774772644, -0.4015365540981293, -1.5704203844070435, 0.5329243540763855, -0.942370593547821, -1.5245076417922974, 1.3326084613800049, 0.9101752042770386, -0.9055973291397095, -0.8009964227676392, -0.3590620160102844, 0.3359496295452118, 1.1440691947937012, -0.11806249618530273, -1.207943320274353, 1.5923572778701782, 1.1296029090881348, 1.124068260192871, 1.6797047853469849, 0.950372576713562, -1.892655372619629, -1.4339921474456787, -0.3639986515045166, 0.7674978375434875, -0.008729065768420696, 0.6406177282333374, 1.1585676670074463, -0.8018815517425537], [0.2093820422887802, 0.6888322830200195, 1.902999997138977, -1.0796351432800293, 0.2618268132209778, -0.2271520495414734, -0.5819028615951538, -0.6719762682914734, -0.03203534707427025, 0.7370111346244812, 1.0214083194732666, 1.17190682888031, -1.0216954946517944, -0.4466184973716736, 2.2637851238250732, -1.0200445652008057, 0.0626915767788887, -1.4972307682037354, 0.657680094242096, -1.7965035438537598, 0.8799716830253601, 1.6583689451217651, 1.1055222749710083, 0.3552260398864746, 0.5957100987434387, 1.5208064317703247, -0.31233808398246765, -0.013904714956879616, -1.4197791814804077, -0.7646453976631165, 0.35718122124671936, -0.4833788275718689, -1.3783047199249268, 1.0329596996307373, -0.8963655233383179, -0.6325464248657227, -0.8403244614601135, -1.619989275932312, 2.4450314044952393, -2.342562198638916, -0.0768815129995346, -0.10563269257545471, 0.08463292568922043, 0.6041708588600159, 1.8209470510482788, 0.46501925587654114, 2.078653573989868, -1.83354651927948, -0.46489471197128296, -2.3881583213806152, -0.5627370476722717, 0.8032186627388, -0.48653966188430786, 1.3713105916976929, 1.693616509437561, -1.3482900857925415, 1.9799710512161255, 0.23565547168254852, 0.44726860523223877, 1.0001646280288696, 0.20961794257164001, -1.1845327615737915, 0.529313325881958, -0.04265187308192253, 0.12350817024707794, 0.06595982611179352, 0.6241220831871033, 0.10559368133544922, 0.275540828704834, 0.5369042158126831, 0.5717282891273499, -1.1914372444152832, -0.7638515830039978, 0.4608525335788727, -0.4162355661392212, -0.5430403351783752, -1.5529475212097168, -1.560945749282837, -0.7013238072395325, 0.8232101798057556, -0.16832397878170013, 0.6840124726295471, 0.4785202741622925, 0.9130313396453857, 1.4313483238220215, -0.1788894236087799, -0.9373488426208496, 0.18567724525928497, -0.7681679725646973, 0.20656169950962067, 0.1942722052335739, 0.8386368751525879, -0.28409165143966675, -1.625649094581604, 0.9375629425048828, -0.6434534192085266, 0.8083195686340332, 0.7614936232566833, 0.20645484328269958, -0.2866832911968231, 2.1432313919067383, 1.7201752662658691, 0.2018708884716034, 1.558354139328003, 0.2939775884151459, -0.330331027507782, -0.23385775089263916, -2.404305934906006, 1.8050813674926758, 0.7896145582199097, -0.34344643354415894, -0.5514686703681946, 0.6239396929740906, 1.563602089881897, -0.12970732152462006, 1.8553717136383057, 0.07037176936864853, -0.06489024311304092, 1.2429509162902832, -0.5946885347366333, 0.5655447840690613, -0.440897673368454, -0.2866838276386261, -1.4321703910827637, -0.9114280343055725, 0.46109598875045776, -1.4903357028961182, 0.6196268200874329], [-0.6246849894523621, -2.9561009407043457, -1.728348731994629, -1.055881381034851, 0.5256050825119019, -0.04394958168268204, -2.039875030517578, 0.6231623888015747, -0.6899824142456055, 0.11910071969032288, -0.6925210952758789, 0.0653863474726677, 0.03838175907731056, 1.0071439743041992, -0.5611307621002197, -0.9854578971862793, -0.5325320959091187, -0.22091837227344513, 0.4200741648674011, -1.4400897026062012, -0.7933651208877563, 1.2535725831985474, -0.21166229248046875, -0.2410253882408142, -0.4213049113750458, -0.6866877675056458, -0.7718992829322815, 0.4627397656440735, 0.8193471431732178, 0.11556816846132278, 0.3453054428100586, -1.1486141681671143, 0.5492669343948364, -0.6209805011749268, 0.7564553022384644, 0.41161492466926575, 1.0750495195388794, 0.530354380607605, 1.129598617553711, -1.6673208475112915, -0.13074542582035065, -0.9519011974334717, -0.7518622875213623, 0.9292863607406616, -1.2774685621261597, -0.1657436639070511, -0.5061001777648926, -1.6072920560836792, 1.1302989721298218, -1.0047744512557983, -0.6502974033355713, 1.1156712770462036, 0.19656376540660858, -0.3446739912033081, -1.54036283493042, 0.17518967390060425, 0.7635700702667236, -1.077826976776123, -0.528290331363678, -1.8425934314727783, -1.5188368558883667, 0.5747544765472412, 0.1596153825521469, 0.6325332522392273, 0.5009214878082275, -2.0589663982391357, 2.0964207649230957, 0.14104455709457397, 1.3787132501602173, 0.6728799939155579, -0.3246528208255768, 1.145310878753662, 0.40715304017066956, -2.6863605976104736, 0.4077479839324951, 1.3234246969223022, -2.0562222003936768, 0.12417405843734741, -1.1974694728851318, -0.021195026114583015, 0.04377591237425804, 0.631206750869751, 0.0597558468580246, -1.0738601684570312, -0.21384519338607788, -0.3066475987434387, -2.295720338821411, -1.8760408163070679, 0.10285986959934235, -1.0075697898864746, -0.11489663273096085, -1.4588104486465454, 0.3957195580005646, -1.0692113637924194, -0.5497905611991882, -0.9636638760566711, -0.8290184140205383, -0.16964152455329895, -0.058924175798892975, -0.6295578479766846, -0.2762806713581085, -1.0305482149124146, 0.28473666310310364, 1.4533462524414062, 1.6215651035308838, -0.2702754735946655, -1.2320137023925781, 0.015329006128013134, 0.3516295552253723, 0.19864781200885773, 0.1776529848575592, 1.2511669397354126, 0.20887631177902222, 0.6698211431503296, 1.4122354984283447, -1.4780230522155762, 0.195454403758049, 0.036726828664541245, 1.3360353708267212, 1.5405033826828003, -1.6206923723220825, -1.1007837057113647, -0.03724588826298714, 2.4732751846313477, 2.019787549972534, -0.6770146489143372, -2.477794885635376, -1.4369322061538696], [0.2971450984477997, -0.6761437654495239, -0.71974778175354, -1.2686474323272705, 0.43261706829071045, -1.248489260673523, 0.6781628727912903, 1.2053172588348389, -1.734352946281433, 0.7438629865646362, -0.9681196808815002, -1.0611693859100342, 0.8336210250854492, -0.2702485918998718, 0.5372074842453003, -0.6370469927787781, 2.2076761722564697, 1.1347063779830933, -0.49242928624153137, -0.14978015422821045, -0.5517497062683105, 1.157293677330017, 0.19114230573177338, -1.3365601301193237, 1.275719404220581, -0.028001047670841217, 1.7887974977493286, 0.45320630073547363, -1.4211459159851074, -3.156428575515747, -0.0618792399764061, -0.2763940095901489, 1.8080534934997559, 0.2631760835647583, -0.5016417503356934, -0.6202940344810486, -1.8429350852966309, 1.1313910484313965, -0.41060060262680054, 1.2316062450408936, -0.886177659034729, 0.5651801228523254, 0.12669648230075836, 1.0567480325698853, 0.18687976896762848, -0.04836627468466759, -1.683240294456482, -0.019161760807037354, 0.8690993189811707, -1.0644837617874146, 1.4725676774978638, 0.6003062129020691, -0.3944469094276428, -0.5611193180084229, 2.69380784034729, -1.1129604578018188, 0.5760061144828796, -0.21736931800842285, -0.04403584450483322, -0.3842185437679291, 0.07822314649820328, -0.4496462941169739, -3.1049413681030273, 0.3772514760494232, 0.5362486243247986, -1.7843754291534424, -0.2544083595275879, 0.33974820375442505, -0.9455527067184448, 1.0785192251205444, -1.596695065498352, 1.5384275913238525, -0.35174256563186646, 0.023654790595173836, -0.6328115463256836, -0.6613070368766785, -0.4765620827674866, -1.4465128183364868, -0.07262590527534485, 0.24138016998767853, -0.8719232082366943, 0.5331794619560242, -0.015630658715963364, -0.8987240195274353, -0.5490732789039612, 2.0254061222076416, -0.8500559329986572, 0.14076527953147888, -1.4734807014465332, 1.324480414390564, 0.5210182666778564, 0.31360793113708496, -2.265977621078491, 0.5465618371963501, 0.05764389410614967, -0.9307091236114502, -1.1140369176864624, 0.2012656182050705, 0.4200170338153839, 1.1060835123062134, -0.5557715892791748, -0.6441503763198853, -0.14725510776042938, 0.14387445151805878, -0.6491245627403259, 0.2046809047460556, -0.4202827215194702, 0.03406607359647751, 2.0493836402893066, 1.1838444471359253, -1.1365069150924683, 1.8499622344970703, -0.06888902932405472, 0.6184632182121277, 1.562589168548584, -1.7513444423675537, 0.5281266570091248, -1.0168882608413696, -0.3640812337398529, -0.2400483936071396, -0.9694293141365051, -0.9001749157905579, 0.30362144112586975, 0.03261013329029083, -0.25934740900993347, -0.0361735038459301, 0.28001531958580017, -0.6932361721992493], [-0.21409012377262115, -0.3730124235153198, -2.1535651683807373, -0.05273216590285301, -1.3334980010986328, 2.4103896617889404, 0.09076917916536331, -1.3720452785491943, -0.007265046704560518, 1.3483206033706665, -0.466360479593277, -0.753539502620697, -0.14881090819835663, -0.22713343799114227, 1.6146844625473022, 0.5230532884597778, -1.5686538219451904, -0.30685457587242126, -1.1056839227676392, 0.2605447471141815, -0.5903225541114807, -0.271584689617157, 1.3062666654586792, 0.6876598596572876, -1.0327377319335938, -0.5656210780143738, 0.3238675892353058, 1.050360918045044, -0.3556990921497345, -0.24032022058963776, -0.6541116833686829, -0.379138320684433, 1.183627724647522, -0.2936241328716278, 1.3433375358581543, 0.48367390036582947, -1.231766939163208, 1.0142409801483154, 1.1622565984725952, -1.3627394437789917, -1.4001094102859497, -1.343499779701233, 0.9563096761703491, -0.3787091076374054, 2.2553794384002686, -0.44571611285209656, -1.0928115844726562, 0.4348619282245636, 0.891187310218811, 0.3111162781715393, 0.5366272926330566, -0.801254153251648, -1.3853824138641357, -0.18164493143558502, -0.17496436834335327, 1.7882301807403564, 2.010002613067627, 0.9597629904747009, -0.8732207417488098, -1.218634843826294, -1.9974274635314941, 0.047593604773283005, 0.10494158416986465, 0.917474627494812, -0.9736865162849426, 1.125768780708313, 0.6031918525695801, -1.5337899923324585, -2.0987906455993652, 0.566375732421875, -0.7965033650398254, -0.7094354629516602, 1.1417204141616821, 0.5990583896636963, 1.2647030353546143, 1.0152909755706787, -0.5317756533622742, 1.5157833099365234, 1.2634022235870361, 0.33292385935783386, -1.9173325300216675, 1.3455595970153809, -1.005810260772705, -0.003988124430179596, 0.6733497977256775, -0.8979270458221436, 0.023176567628979683, -0.9656083583831787, 0.12543341517448425, -0.7024132609367371, 1.1016895771026611, 0.4571228623390198, -2.409026861190796, 0.6761590242385864, -1.5568112134933472, -1.831783652305603, 0.8890433311462402, -1.0200316905975342, -0.4143337905406952, -0.45379137992858887, 0.18773877620697021, 0.06757992506027222, 0.27704113721847534, -0.4087955355644226, 0.766008734703064, 0.5910008549690247, -0.013764988631010056, 1.838842749595642, 0.04407487064599991, 1.348944067955017, 0.5624471306800842, -1.1682049036026, -2.037888765335083, -0.4072304666042328, -0.7809930443763733, -0.9305076599121094, 0.7028926014900208, 1.311948299407959, 0.8092638254165649, -0.9295450448989868, 0.3262002170085907, 1.9494057893753052, -0.3768270015716553, -2.386042594909668, 0.6537052989006042, -0.991737961769104, -2.0391156673431396, 0.4598441421985626], [-0.7973616719245911, -0.9278777837753296, 0.6491450667381287, 0.10501212626695633, 0.2814001441001892, -0.6149962544441223, 0.6747648119926453, 1.8364278078079224, -1.1035728454589844, 0.8071211576461792, 1.1589233875274658, 0.1306152492761612, -0.06433935463428497, 1.2349498271942139, -0.3335024416446686, 0.37555742263793945, 1.1791324615478516, 1.5968660116195679, 0.8342259526252747, 0.256600022315979, 0.834682047367096, -0.2534593939781189, -0.11931265145540237, 0.3284345269203186, 0.5801201462745667, 0.7409899830818176, -2.2130391597747803, -2.766707181930542, 0.11463487893342972, -0.25056397914886475, 0.21766217052936554, 0.9100053906440735, 0.34461650252342224, 1.1456730365753174, -0.7174646258354187, -0.5732777714729309, 1.422972559928894, -0.24739409983158112, -0.3831256628036499, -0.39136484265327454, 1.2956724166870117, -0.8549808859825134, -0.6557433605194092, 1.2669719457626343, 1.2696583271026611, -0.11940848082304001, 0.6288303732872009, 1.7850197553634644, 1.9082897901535034, 0.45526009798049927, -0.3256683051586151, -1.3836703300476074, -0.7988153100013733, -0.732756495475769, -1.8559606075286865, -0.20320755243301392, -0.32344362139701843, 0.07996252924203873, -1.4395147562026978, 0.9962015151977539, 1.074485182762146, 0.8293875455856323, -1.84060537815094, 0.42156779766082764, -1.364363193511963, -0.5913557410240173, 0.024235043674707413, 1.495041847229004, -1.338923454284668, -0.22490739822387695, -0.08369883894920349, -1.0464978218078613, 1.461958408355713, 1.0691955089569092, 0.8725233674049377, 0.7501778602600098, -0.4298323094844818, -0.16375978291034698, -0.7996968030929565, -0.8567664623260498, 0.5402581691741943, 0.6519982814788818, -1.089150071144104, -2.4246764183044434, 0.45277974009513855, -0.15660136938095093, 0.6381690502166748, -0.32738956809043884, 0.1270274519920349, -0.16119050979614258, 0.33685559034347534, -0.15025652945041656, 0.46216800808906555, -1.1374194622039795, -0.14927224814891815, 1.4588849544525146, 0.8993914127349854, -1.6126365661621094, -0.360182523727417, -0.3610803186893463, -2.18917179107666, 2.8925631046295166, -0.7189017534255981, -2.532945394515991, -0.20522131025791168, -0.47143417596817017, -1.3691949844360352, -0.12257906794548035, -0.9627601504325867, -1.9123613834381104, -0.5867320895195007, -0.24983417987823486, 1.0318821668624878, 0.8735132217407227, 0.5818920135498047, 1.7296737432479858, -0.327400267124176, 1.0717480182647705, -1.3740336894989014, 1.6985162496566772, 1.8611111640930176, 1.038332462310791, -0.3003861904144287, 0.7447733283042908, -0.17790938913822174, -0.878118634223938, 0.05661074444651604, -1.1402032375335693], [0.21365080773830414, -0.9363885521888733, 0.06864796578884125, 0.2081851065158844, -0.2911786735057831, -0.4088402986526489, -0.7608641982078552, 0.32538628578186035, -0.04769878461956978, -0.38323938846588135, -0.1366586536169052, 2.097231388092041, 0.1985437422990799, 1.0108004808425903, -1.1435086727142334, -0.8261528611183167, -0.9259807467460632, -0.3708863854408264, 2.206575393676758, -0.06137194111943245, 0.9159995913505554, -0.8298834562301636, -0.86086106300354, -1.2760547399520874, 0.1562584936618805, -0.9893848299980164, 0.5132121443748474, -0.021320391446352005, 0.21278850734233856, -0.058377135545015335, 0.3113943636417389, -1.0587397813796997, 0.1541525423526764, 0.38862425088882446, -0.3110160827636719, 0.4203874170780182, -0.21522822976112366, -0.690232515335083, 0.7283750772476196, 0.8299908638000488, -1.1944242715835571, -0.9748319983482361, 1.3471393585205078, -2.407397508621216, -0.5665455460548401, -0.040179505944252014, 0.3516104221343994, -0.38603490591049194, -1.5251175165176392, 0.19078484177589417, -0.9120528101921082, 0.5514050722122192, 0.6715399026870728, -0.40699687600135803, 0.3170638382434845, 0.10151305049657822, 0.5251103043556213, -1.109735131263733, 0.6247642636299133, -0.7401387691497803, 1.4773261547088623, 0.5075405240058899, 1.2386220693588257, 0.035725489258766174, 0.03812336549162865, 1.139603614807129, 0.1649179458618164, -1.304731011390686, 0.12245075404644012, 0.21569418907165527, -1.0741817951202393, -1.7483644485473633, -0.3753942847251892, -0.7416713833808899, 0.7666995525360107, -0.7629594802856445, 1.242687463760376, 0.299618124961853, -0.3643641471862793, 1.038059115409851, -1.0982286930084229, 1.7350306510925293, 1.20091712474823, -1.4177298545837402, 1.3869770765304565, -1.3161425590515137, -0.8579936027526855, -0.12188854813575745, -1.1510672569274902, 1.2407938241958618, 0.295081228017807, 0.7755509614944458, 2.6496269702911377, 0.9155728220939636, 1.112184762954712, 0.18503719568252563, -0.6971719861030579, 0.6389463543891907, -1.3912497758865356, 0.8247156739234924, 0.07857012003660202, 0.26866739988327026, 0.2117946296930313, -0.4685438871383667, -1.2957944869995117, -1.5274944305419922, 0.9128754138946533, 0.5527074337005615, -0.16204623878002167, -1.089636206626892, -0.4859040677547455, 1.502202033996582, 0.4075164198875427, -0.6961768269538879, 0.5207324028015137, -0.8532535433769226, 0.6366468071937561, -0.7314636707305908, -0.37062889337539673, 1.1783647537231445, 0.8298256993293762, -1.369714379310608, 1.2636798620224, -0.1635832041501999, 0.2979133427143097, -0.2589019536972046, 1.194420337677002, 0.03417646512389183], [-0.5065619349479675, -0.3024793863296509, 0.8520359992980957, -2.6328587532043457, -0.0646398738026619, -1.1308053731918335, 1.1581556797027588, 0.9154115319252014, -1.3465394973754883, -1.5494322776794434, -1.2263929843902588, -1.1303558349609375, -0.5768336653709412, -0.06176992133259773, 0.679193913936615, -0.31589475274086, -0.4750059247016907, -1.311238408088684, 0.8411339521408081, -0.4057596027851105, 1.243553638458252, 0.36837267875671387, 0.530860185623169, -1.8810919523239136, 0.17424975335597992, 0.011388067156076431, 0.360578715801239, -1.2686147689819336, 0.18587726354599, 0.5736208558082581, -0.06268496811389923, -1.5470799207687378, 0.6820282340049744, 0.8209022879600525, 1.5883815288543701, -0.8431447148323059, -0.41063785552978516, -1.3623102903366089, -0.36277303099632263, -0.9412590265274048, -0.688189685344696, 0.4455285370349884, 1.679552674293518, 0.774949848651886, 0.0020421885419636965, 0.5651196241378784, -0.010890768840909004, 0.1166715919971466, -1.7419370412826538, 0.8382942080497742, 0.19878023862838745, 0.2974987328052521, 1.4160826206207275, 0.8163318037986755, 1.8069791793823242, -0.9464813470840454, 0.5000841617584229, 0.39032191038131714, 0.9503348469734192, 0.746442973613739, -1.1609796285629272, 1.1253917217254639, -0.8400513529777527, -1.1763707399368286, -0.8311179876327515, 1.6011167764663696, 0.6085337400436401, 0.36046919226646423, 2.3188953399658203, -1.282637119293213, 0.5407384037971497, 0.9745416641235352, -0.5086112022399902, 0.06772937625646591, -1.6761529445648193, 0.48400381207466125, -0.7258495092391968, 1.2315312623977661, 2.1996243000030518, 0.06106723099946976, 0.6527523398399353, 0.45394167304039, 0.9620591998100281, 1.7324529886245728, -1.3974641561508179, -0.28916218876838684, -0.9267811179161072, -0.1399228572845459, -0.2842878997325897, -1.2837097644805908, -1.0959324836730957, -0.19702030718326569, 0.9571974277496338, 1.0653769969940186, -0.882905125617981, -0.26685693860054016, 0.39626702666282654, 0.5806485414505005, -0.3394523859024048, -1.191702127456665, -1.727617859840393, 1.6093348264694214, -0.3547222316265106, -0.9144349098205566, -0.05345665663480759, -0.9156510233879089, -0.5940700173377991, 0.31149017810821533, 0.21315225958824158, -0.9419509172439575, 1.2140417098999023, -1.277095079421997, -0.3761197626590729, -0.8638750910758972, 0.043566592037677765, -0.24331143498420715, 0.3832482099533081, 1.1529515981674194, 1.6465741395950317, 0.30000221729278564, 0.8026558756828308, -0.015680687502026558, -0.04103328660130501, 1.5891313552856445, 0.22810710966587067, -0.8549426794052124, 0.4825262129306793, 1.329406976699829], [1.027220606803894, 1.7301228046417236, -0.34559834003448486, -0.2633266746997833, -1.0109480619430542, 1.0994815826416016, 0.06319981813430786, 1.8494668006896973, 1.3274673223495483, 0.21385866403579712, -0.2286866307258606, -1.0304542779922485, 2.6107916831970215, -0.238671213388443, 1.2195554971694946, -0.8297210335731506, 0.8959353566169739, 0.8120238780975342, -0.49674639105796814, 0.9630206227302551, 1.1999719142913818, -0.3674612045288086, 0.2166913002729416, -0.2631010115146637, -0.8829414248466492, -1.5680220127105713, 0.1467902958393097, 1.474554181098938, -1.5819683074951172, 0.7721244096755981, -0.03886409103870392, 1.5074259042739868, 0.3365227282047272, -0.9073224067687988, 0.7447463870048523, -1.577531337738037, 0.5864871144294739, 0.05720837041735649, -1.4978028535842896, -1.4662846326828003, 0.8892354965209961, 0.05315522104501724, -2.1327388286590576, -1.0944287776947021, 0.4271831512451172, -0.09923111647367477, 0.9236114025115967, 0.1257735639810562, 1.11435866355896, -1.4783391952514648, -1.1207257509231567, 1.2599996328353882, -1.2245622873306274, -1.166947364807129, 0.5628848671913147, -0.18203628063201904, 1.3150194883346558, -0.26072150468826294, 1.5170165300369263, -0.7327746152877808, -0.967338502407074, 0.4431706666946411, -1.43757164478302, 1.065110683441162, 1.3566519021987915, -0.5980780124664307, -0.6603139638900757, -1.0932857990264893, -1.4373503923416138, 0.16826112568378448, 0.41521474719047546, 0.8971379399299622, 0.42097046971321106, -0.07339683920145035, 0.3464873433113098, 0.1651284098625183, -0.06762846559286118, 0.3322174847126007, 0.35097160935401917, -1.1772382259368896, 0.1518407016992569, 3.424546241760254, 0.7619858980178833, 0.8204770088195801, 0.14973171055316925, -0.09592833369970322, -0.2599170506000519, -0.6255000829696655, 0.24317488074302673, -0.03840929642319679, -1.0943446159362793, -0.7980312705039978, 1.8409916162490845, -0.02133948542177677, -1.3991968631744385, 1.7242364883422852, 0.6616572141647339, -0.0771380290389061, -0.12387022376060486, -2.1975154876708984, 0.8155677914619446, -0.9282603859901428, -2.1072633266448975, -2.3488776683807373, 0.019529668614268303, -1.1277917623519897, -0.938786506652832, -0.2852833569049835, -1.1893606185913086, 0.7604009509086609, -1.0781595706939697, -0.550190806388855, -0.44935664534568787, 0.28374916315078735, 0.15553666651248932, 1.165260672569275, 0.9806491732597351, 0.048026036471128464, 0.40407687425613403, 0.48078039288520813, 0.20557627081871033, 1.1622059345245361, -0.1635674685239792, -1.1979674100875854, -1.1521564722061157, -2.0382814407348633, -0.6906809210777283, 2.3334527015686035], [1.684230923652649, -0.5876181125640869, 0.7631697654724121, 1.0785142183303833, 1.150355339050293, -0.3547542095184326, 0.24675889313220978, -0.25303202867507935, 0.6037725210189819, 1.0986195802688599, -0.9196267127990723, -0.9423813819885254, 1.679168462753296, 1.0018244981765747, 0.4854460060596466, -0.5472642779350281, -0.6111990213394165, 0.1759767383337021, -1.1599138975143433, 0.7487642765045166, 0.7696691751480103, -0.8765849471092224, 0.7039768695831299, -0.18548819422721863, 0.9369907975196838, -1.5481809377670288, -0.1512017697095871, -0.2100934237241745, -1.419253945350647, -0.41555094718933105, 1.269731879234314, 1.0553160905838013, 0.7093477249145508, 2.330687999725342, -1.4964765310287476, -0.34602090716362, -0.2689455449581146, -0.5926432609558105, -1.8359588384628296, 1.569271206855774, 1.417257308959961, 0.5024587512016296, 0.9541721940040588, 0.7802327871322632, -0.16833660006523132, -1.2843348979949951, 0.1772257387638092, -0.1506900191307068, -0.22270382940769196, -1.7101900577545166, -0.5831116437911987, -0.8230295777320862, 0.4643067419528961, -0.7439783811569214, -0.8587275743484497, 0.24796317517757416, 0.7978287935256958, -0.6895995736122131, 0.3962307274341583, -1.6630663871765137, -0.2857015132904053, -0.28069183230400085, 0.5953629016876221, -1.0314921140670776, 0.38079139590263367, -0.7947635054588318, -0.0035220065619796515, -1.2985316514968872, -0.11121950298547745, 1.0948824882507324, -1.0238789319992065, 0.898669958114624, 1.9069857597351074, 1.3713653087615967, -1.2684246301651, 0.3176015019416809, -0.8159079551696777, -0.6554272770881653, 0.2637793719768524, 1.0340261459350586, 0.649355411529541, 1.3048458099365234, 0.2894676923751831, -2.010361909866333, -1.3369742631912231, 1.1196821928024292, 0.5690990090370178, 0.6597016453742981, -1.0160839557647705, -0.6917330622673035, 0.06214065104722977, 1.3951088190078735, -2.2858076095581055, 0.47243058681488037, 0.740077555179596, -0.18247312307357788, -0.5681853294372559, 1.233309268951416, -0.9196562767028809, 0.101236991584301, 0.6411806344985962, -0.42539069056510925, 1.3914071321487427, 1.54865300655365, 0.3124178647994995, 0.560121476650238, -1.4682788848876953, 0.7813104391098022, -1.1563045978546143, -0.1951206773519516, -0.6573917865753174, -2.3869619369506836, 0.39637500047683716, 0.4922325909137726, -0.1090352013707161, -1.0362529754638672, -0.22967472672462463, -1.3549021482467651, 2.1005635261535645, 0.44912803173065186, 2.4793853759765625, -0.5752666592597961, -1.4330936670303345, 0.005707933567464352, 0.09736598283052444, -1.583479642868042, -0.6063992977142334, -0.8874954581260681], [0.9648095965385437, -0.2949208617210388, 1.287294864654541, -0.791971743106842, 0.34913378953933716, -0.20917731523513794, -0.5044708847999573, 0.16700029373168945, -0.9490258097648621, 0.3087582588195801, 0.8026031255722046, 0.6036773920059204, 1.1904730796813965, -1.1836680173873901, 1.6909277439117432, -0.4697629511356354, 1.5072007179260254, 2.2194759845733643, 0.32584452629089355, 0.331037312746048, -0.7652121186256409, 1.073730707168579, -0.037393711507320404, -0.1495465785264969, 1.1237828731536865, -0.6727063059806824, -0.7538083791732788, -1.6924622058868408, 0.6084914207458496, 0.3165377974510193, -0.36503997445106506, 0.075379878282547, 1.002734661102295, -0.24007244408130646, 0.7427836656570435, -0.38940805196762085, 0.518052875995636, -2.5261361598968506, -0.24014140665531158, 0.22367709875106812, 0.9566092491149902, -1.089438796043396, -0.8222568035125732, -0.272165983915329, -0.23375184834003448, -0.5293434858322144, 0.7319176197052002, 0.5569950938224792, 1.0770456790924072, 0.27063360810279846, -1.019597053527832, 0.11429926753044128, 0.017176317051053047, -0.1126563772559166, 1.6396043300628662, 0.6446216106414795, 1.1426280736923218, 0.7973176836967468, -0.6093605160713196, -1.0360431671142578, -0.1191740334033966, 0.8897333145141602, -0.24724382162094116, 0.28405165672302246, -0.00972782913595438, -0.8765720725059509, -0.4309171140193939, 0.5022003650665283, 0.20391634106636047, 0.8619138598442078, 0.8798220157623291, 0.04766269028186798, -0.9116744995117188, -1.2792243957519531, -0.6222596168518066, 0.34283971786499023, -1.6029618978500366, 1.3264281749725342, -0.4756699204444885, -1.725083351135254, 0.174315944314003, -2.270921468734741, 0.33338963985443115, -0.5447578430175781, 0.23649761080741882, -1.1426129341125488, -0.5274553894996643, 0.15010179579257965, 0.12038663029670715, 1.5737135410308838, 1.2145789861679077, 0.24343359470367432, -0.4502533972263336, -0.7276486158370972, -2.015972375869751, 0.221767395734787, 1.250754714012146, 0.8012605905532837, 2.1285173892974854, 0.6170642971992493, -0.17607201635837555, 0.7867603898048401, 0.4072720408439636, -0.2623313367366791, -0.6844518780708313, -0.8334383368492126, -0.6849116086959839, 0.14211004972457886, 0.20015381276607513, 0.41604670882225037, 1.2961430549621582, 0.33170294761657715, -0.14009897410869598, -0.7255426645278931, 0.28589892387390137, 0.212804913520813, 1.2344708442687988, -0.22188879549503326, 2.2951440811157227, -0.3325732350349426, -0.13559593260288239, 0.730519711971283, -0.008544174022972584, -1.1136887073516846, -0.5976006388664246, -0.2982213795185089, -0.630843997001648, 0.5359531044960022], [0.10410308837890625, 0.41805511713027954, -1.2081586122512817, 0.27646028995513916, -0.05768362805247307, 0.6584112644195557, -0.1537162810564041, -0.5215761661529541, 0.21244022250175476, -0.34129634499549866, 0.42631202936172485, -0.031009508296847343, -0.6947131156921387, 0.8315953016281128, -1.077690839767456, 0.09538283199071884, -0.35508501529693604, 0.595214307308197, 0.5196414589881897, -0.20037993788719177, 0.8416836261749268, -0.3211188316345215, 1.1166023015975952, -0.595043957233429, -2.341667890548706, 0.033363547176122665, -1.0709202289581299, -1.653079628944397, -0.48091939091682434, -0.36889731884002686, 0.5192261934280396, 0.06277373433113098, 0.9323911666870117, 0.8524130582809448, 0.8010227084159851, -2.5216012001037598, 1.1140036582946777, 0.46953266859054565, 0.6799096465110779, -1.155761957168579, 0.7958563566207886, -1.656325340270996, -0.2395617663860321, 0.01693783327937126, -0.7645432949066162, 0.8185005784034729, -0.9539774656295776, 1.2195262908935547, -0.2542710602283478, -2.2557315826416016, 0.5111193060874939, 0.3167586624622345, -0.36863237619400024, 0.3552379608154297, 1.2428488731384277, -0.12673752009868622, -0.7079582810401917, 1.7520779371261597, 0.3023397922515869, -0.6745935678482056, -1.6234465837478638, -0.10968201607465744, 1.1823232173919678, -0.7494341731071472, 1.5231043100357056, 2.1283795833587646, 0.35996779799461365, 1.0329197645187378, 2.788414716720581, -0.5795786380767822, -1.7122080326080322, -2.345391035079956, 0.4810921847820282, -2.361109733581543, -0.23880895972251892, -0.981644332408905, 2.2614822387695312, -1.7115696668624878, -1.884422779083252, 0.5432775020599365, 0.7834085822105408, 0.6868998408317566, 1.8727608919143677, -1.0688501596450806, -0.661939263343811, -0.7052001357078552, -0.4482460916042328, -1.727548360824585, -0.47925612330436707, -0.7065717577934265, -0.027886228635907173, -0.7801045775413513, -0.7883492708206177, 2.1750869750976562, -0.9782225489616394, 0.33935850858688354, -1.6393482685089111, -0.11643866449594498, 1.2983063459396362, 1.522130012512207, -1.9361766576766968, -1.9755414724349976, -1.573994517326355, 0.6330022215843201, -1.277153730392456, 1.0165929794311523, 0.9461311101913452, 0.11739934235811234, -0.921757698059082, -0.05729811638593674, -1.72416090965271, -0.34512224793434143, 0.29519328474998474, -0.556522011756897, 0.14935694634914398, -0.5763130784034729, 0.7762840986251831, 0.05141830071806908, 0.42533621191978455, 0.856313169002533, -2.0955698490142822, -0.08718381822109222, -0.3078761100769043, 0.19452665746212006, -1.79063880443573, 0.46951717138290405, 0.3452930748462677, -0.956649899482727], [-1.0649725198745728, -1.0811667442321777, 1.5642937421798706, -0.18570442497730255, 0.2802399694919586, -0.5872596502304077, -0.07899268716573715, -0.6370261311531067, -0.4025396704673767, 1.015782356262207, -1.3476676940917969, 0.6200904846191406, -0.048138782382011414, -0.667415976524353, 0.18226255476474762, 1.3191378116607666, -0.8925489783287048, 0.7743829488754272, 0.1991388499736786, 0.4390513598918915, 2.25982928276062, 0.0729527622461319, -1.1830847263336182, -1.0851843357086182, 0.30583810806274414, -0.4943302571773529, -1.1101300716400146, -0.07760310173034668, 0.4049929082393646, 2.035799741744995, 1.4186912775039673, 1.7344717979431152, -0.49753978848457336, -0.5845958590507507, 0.04151991009712219, -0.25757917761802673, 1.87935471534729, 0.5671845078468323, 0.6960716247558594, -1.816041350364685, 0.9150491952896118, 1.1635193824768066, -0.9474005699157715, 0.7598690390586853, -0.7697198390960693, -0.8552563786506653, -0.38709813356399536, 0.6449278593063354, 0.052494682371616364, 0.8790873289108276, -0.5390637516975403, -0.6965205669403076, -1.9588662385940552, 0.8217646479606628, -0.18857726454734802, -0.18015676736831665, -1.0652518272399902, -2.129948616027832, 0.5115225911140442, -1.8781485557556152, -0.39599987864494324, 0.4544714391231537, -2.276698589324951, 0.31451621651649475, -1.3246186971664429, -1.4845006465911865, 0.25935861468315125, 1.4269676208496094, 0.9486919045448303, 0.2302495688199997, -1.6465539932250977, 0.6549002528190613, -0.5377961993217468, 0.4857359528541565, -0.10677142441272736, -0.17352566123008728, -0.6614877581596375, 0.21328303217887878, -0.18122662603855133, -1.013819694519043, -1.7110341787338257, -0.924430251121521, 1.4167522192001343, 0.304799348115921, -1.5349456071853638, 2.4647395610809326, -2.0523746013641357, -0.5719922780990601, 0.9940799474716187, -0.18875250220298767, -0.43475955724716187, 0.23087556660175323, 0.32064470648765564, 0.04668625071644783, -1.0548449754714966, -1.30502450466156, 0.13015125691890717, 0.6261842846870422, 0.9907397627830505, 0.7978368401527405, -1.955872654914856, -1.338505744934082, 1.4773274660110474, 1.8460220098495483, 1.094017505645752, -1.4135950803756714, 0.9980933666229248, -1.5189661979675293, -0.3823845088481903, -1.015572190284729, 0.6439841389656067, -0.12217757105827332, 1.1631346940994263, -0.8215912580490112, -0.8673005700111389, 0.920813262462616, -0.59124356508255, 0.35500937700271606, -1.442662239074707, -0.9424843192100525, 1.8039076328277588, 0.6034761071205139, -0.4129537045955658, -0.9704292416572571, 0.8496685028076172, -1.0442731380462646, 0.2608262002468109, 1.4625376462936401], [-0.11266493797302246, -0.9998359680175781, 1.1932274103164673, 0.5964768528938293, -0.012968736700713634, 0.18964321911334991, -1.031282901763916, -1.4139368534088135, -2.1579737663269043, -0.8913447856903076, 0.2882952094078064, 0.39849668741226196, -0.01959003508090973, -1.6830390691757202, 0.3829966187477112, 1.2434308528900146, 0.03902226313948631, 0.38216760754585266, 1.6202285289764404, -0.5879173278808594, -0.8229283690452576, 1.3122897148132324, -0.1654539406299591, -1.4648818969726562, 0.09495710581541061, 1.0541765689849854, -0.8649938702583313, -0.41207849979400635, 1.2672250270843506, 0.04881976917386055, 2.4099955558776855, 0.5179744362831116, -0.32646575570106506, -0.6653891205787659, -0.2585446238517761, 0.03691617771983147, -1.0280778408050537, 0.0462300069630146, 0.16226054728031158, 0.809756875038147, -1.9555180072784424, 1.1639022827148438, -0.4209774136543274, -1.4169297218322754, 0.21626362204551697, 0.681405246257782, -1.2160152196884155, -1.3844351768493652, -1.3880864381790161, -0.8390698432922363, -0.0653114914894104, 1.323264718055725, -0.21025502681732178, 1.0217341184616089, 1.8641995191574097, 0.8170140981674194, 0.5815935134887695, 1.8245655298233032, 0.7685457468032837, 0.313951700925827, -1.0220987796783447, 2.0809051990509033, 0.31932052969932556, 0.4135103225708008, -0.736309826374054, -0.8924290537834167, 0.4526168406009674, -0.8565366864204407, -0.07905468344688416, 1.0795834064483643, 0.3804086446762085, -0.22044752538204193, -0.11367377638816833, 2.1965255737304688, 0.622355580329895, 1.8610420227050781, 1.1695584058761597, 0.8516213297843933, -0.7040796279907227, 1.272274136543274, 0.5731298327445984, -1.8962604999542236, 0.6437993049621582, 1.1545968055725098, 2.9864418506622314, 0.7928082346916199, -0.2976624071598053, 1.049790859222412, 0.21640750765800476, 0.5245945453643799, 0.6609926223754883, 0.7411110997200012, -1.3381017446517944, -1.1662101745605469, -0.036579493433237076, 0.7104952931404114, -0.8444367051124573, -0.7686542272567749, -1.608546495437622, 1.5509397983551025, 0.0008953072829172015, -0.7478616833686829, -1.1116775274276733, -0.40366116166114807, -0.8612184524536133, 0.348426878452301, -0.9391244649887085, 0.3326989710330963, -2.6506783962249756, 0.041402265429496765, -2.4705843925476074, 1.076810598373413, 0.8109648823738098, -1.0534400939941406, -0.3534112572669983, -0.4148830771446228, -0.3385302722454071, -1.2554128170013428, -0.7061460614204407, -0.22433629631996155, 0.42840251326560974, 0.25598546862602234, 0.2045057713985443, -0.2894718647003174, -0.6626773476600647, -0.2668885886669159, 0.9368427395820618, -1.1005069017410278], [-0.138604536652565, 0.24466879665851593, -2.3321008682250977, -1.2631800174713135, -0.005964992102235556, 1.054508924484253, -0.7750702500343323, 0.5629662275314331, -0.03511727973818779, 0.2390766143798828, -2.8923566341400146, 0.5444185137748718, 0.425959050655365, 0.10280939191579819, 1.1949408054351807, 1.1383883953094482, -0.26985329389572144, 0.8600215911865234, -0.8701089024543762, 1.7286899089813232, -0.8177950978279114, 0.059645917266607285, -2.185570478439331, 0.4277576804161072, -0.31016287207603455, 0.41092631220817566, 0.30602678656578064, 0.4910082221031189, -1.3464449644088745, 0.4867147207260132, 1.66763174533844, 0.2705509662628174, 0.49988189339637756, 1.1152105331420898, -0.9991695284843445, 2.172443389892578, 0.5913307070732117, -0.18712739646434784, -0.15851183235645294, -1.0826908349990845, 1.5905873775482178, 1.1969138383865356, 0.1469859927892685, 1.6915266513824463, -0.36673903465270996, -0.2565034031867981, 1.293479323387146, -0.9447799921035767, -0.24885542690753937, 0.25817373394966125, -1.0687750577926636, -0.0804118663072586, -0.3777044117450714, 0.4105760455131531, 0.9107628464698792, 1.9190938472747803, -0.24784024059772491, 1.4685792922973633, 0.7796817421913147, 0.2131599485874176, -1.3803991079330444, 0.9684088230133057, -0.17551544308662415, -1.5685175657272339, -1.4186789989471436, 0.0047548082657158375, -1.217521071434021, 1.545499324798584, 0.10198305547237396, 1.267946481704712, 0.012773212976753712, 1.2705779075622559, 0.6739628314971924, -0.6799032688140869, -1.0252383947372437, -0.9848918914794922, -2.590257406234741, 0.14563493430614471, -0.9541804790496826, 0.09487801790237427, 0.22928455471992493, 0.5986731648445129, -0.5562917590141296, -0.3318266272544861, -0.1121087446808815, -0.5658966898918152, 0.7528634071350098, 1.523489236831665, -1.1637459993362427, 0.8083906173706055, 0.5443669557571411, -1.3365626335144043, -0.3482896089553833, 0.8069174885749817, 1.2722244262695312, 0.8219147324562073, -2.4735159873962402, -0.06142328307032585, 0.1585487276315689, 1.063136100769043, -0.050159282982349396, -0.40049031376838684, 0.007759995758533478, 0.9465290904045105, 0.9085466265678406, -0.2414843738079071, -2.115385055541992, -0.7473675608634949, 0.3022412061691284, 1.1551570892333984, -0.8531376123428345, 1.0266250371932983, 1.7777483463287354, -0.07357532531023026, -0.853025496006012, 0.2563219368457794, 0.7490077614784241, 0.3221689760684967, 0.561660885810852, -0.8680513501167297, -0.6277116537094116, 0.03182026371359825, -0.7602038979530334, 0.6202654838562012, -0.5410257577896118, 0.9414142966270447, 0.22411178052425385, -0.7941418290138245], [-0.3453042507171631, -2.066537380218506, -0.39794138073921204, -0.40141308307647705, -1.3958680629730225, 1.9800193309783936, -0.8540054559707642, 0.7401980757713318, -0.44399991631507874, -0.2106654793024063, 1.1012245416641235, -1.339676856994629, -1.1987576484680176, -0.12868502736091614, 0.36036545038223267, 0.6032792925834656, -2.1407153606414795, 1.5500420331954956, -0.08065851032733917, 1.0282528400421143, -1.0107780694961548, 0.09085436165332794, 0.16658955812454224, 1.2468420267105103, 0.05545080080628395, 1.58084237575531, 0.8724126219749451, -0.910407304763794, -0.5244220495223999, -1.4188061952590942, -0.203379824757576, 1.4226678609848022, 0.08927299082279205, -0.8960091471672058, -0.3268681764602661, -0.86748868227005, 0.05222431942820549, -0.5087733864784241, 0.016343364492058754, 0.2954300045967102, -2.783320426940918, -2.2297353744506836, -0.6857569813728333, -0.7482712864875793, -0.3149653673171997, -0.6109165549278259, -0.6289733052253723, 0.30567270517349243, -2.4139046669006348, -0.7910630702972412, -0.46899858117103577, -0.9158639311790466, 0.09895915538072586, 1.7425941228866577, 1.7839561700820923, 2.3032798767089844, -0.8288462162017822, 0.596987247467041, -1.34361732006073, -1.370964527130127, 0.12282045185565948, -1.8681308031082153, 1.2113038301467896, 1.7996772527694702, -0.04865088313817978, -0.11496109515428543, 0.3999597728252411, 1.1576919555664062, 0.21129678189754486, 0.2375165969133377, 0.06957656890153885, 0.18446171283721924, 0.1702234447002411, 0.9081427454948425, 1.6041418313980103, 0.009079594165086746, -0.9712090492248535, -2.591931104660034, -0.5574869513511658, 1.745557427406311, -0.5742234587669373, -0.7592790126800537, 1.4965745210647583, -0.29713162779808044, 2.5149266719818115, -0.9975735545158386, 1.3298249244689941, -0.5223377346992493, -0.8699545860290527, -0.1539228856563568, -0.34042221307754517, 0.7300446033477783, -0.014249837026000023, 2.316957473754883, 0.6083283424377441, 0.3597201704978943, -1.1692593097686768, -0.34649258852005005, -1.5272316932678223, 0.7385073900222778, -0.35365739464759827, -0.11206880956888199, 0.5467347502708435, -0.19033117592334747, -0.3097882568836212, -0.18900884687900543, 1.5886069536209106, -0.8011110424995422, -0.20739084482192993, -0.88011234998703, 1.0949487686157227, -0.30516576766967773, -0.005849387496709824, -0.6003281474113464, -1.1391619443893433, -0.3531228005886078, 0.7150944471359253, -0.10535537451505661, 0.4881172180175781, 0.3312511742115021, -1.4062812328338623, -0.7468155026435852, -0.16519346833229065, 0.38593897223472595, 0.3085475265979767, -3.005614995956421, -1.0321050882339478, -1.6058413982391357], [-1.0852556228637695, -0.6176642775535583, 0.7110692262649536, -1.613431692123413, 0.8543172478675842, 1.7604576349258423, 0.11322443187236786, 0.2953507900238037, 0.898113489151001, 0.7256795763969421, 0.3876948654651642, -1.0353752374649048, 1.8476060628890991, 1.354678750038147, 0.33314600586891174, 1.4629942178726196, -0.651732325553894, 1.394707202911377, 1.5846810340881348, -2.1895699501037598, 0.5134441256523132, -0.37226709723472595, -1.105623483657837, -1.6869332790374756, -0.24742691218852997, 0.7263729572296143, 0.4515421390533447, 0.010515106841921806, -1.5375510454177856, 1.0677980184555054, 0.6946443319320679, -0.31017425656318665, 1.7760257720947266, 0.3369581997394562, -1.1228665113449097, -1.3788926601409912, 1.7467713356018066, 0.1588914394378662, -0.14383253455162048, 1.6217522621154785, -0.3041211664676666, -2.8789806365966797, 2.3579347133636475, -0.47485676407814026, -1.3848052024841309, 0.7672351002693176, 0.6028741598129272, 0.5014849901199341, 0.5919272899627686, 1.0724323987960815, 2.0816104412078857, 0.8895391821861267, -0.8704317212104797, -0.15766185522079468, -1.3147085905075073, -0.8788931369781494, 0.5710614323616028, 1.725844144821167, 1.4732683897018433, -1.5452768802642822, 1.155573844909668, 0.8685577511787415, -0.5193073153495789, -0.13472197949886322, 0.6210182309150696, 0.5255297422409058, -0.3061905801296234, -1.2044315338134766, 0.6241673827171326, -2.5662145614624023, 0.2948221266269684, -2.1005687713623047, 0.8740026950836182, 0.8952987194061279, -1.8489806652069092, -0.2161017507314682, -0.38315820693969727, -1.932579755783081, 0.6432999968528748, -1.9729366302490234, 0.49626219272613525, 1.1789803504943848, 0.6615100502967834, -0.012443185783922672, -1.2994420528411865, -0.6548580527305603, -1.298583984375, 1.7270344495773315, -0.7709770202636719, 1.2767995595932007, -0.4067215621471405, -1.0071531534194946, -0.31383025646209717, -2.588301181793213, 0.3330395519733429, -0.035698652267456055, -1.2695934772491455, 0.2048235684633255, 0.5196940898895264, -0.1581037938594818, 1.3961517810821533, -0.517701268196106, 0.9571896195411682, -0.8656461834907532, 0.24966885149478912, -2.5551016330718994, 1.5905451774597168, 1.5270568132400513, 0.6465771794319153, 1.2657300233840942, 0.2325575202703476, 0.18359141051769257, -0.28992271423339844, 1.2233352661132812, -1.8461570739746094, -1.6416484117507935, 2.402445077896118, 0.6224996447563171, 1.5386725664138794, 0.8664882183074951, -0.23849016427993774, -0.6712965369224548, -1.0160866975784302, -0.2039766162633896, -0.09412355720996857, 0.737891674041748, -0.4972713887691498, 1.0784262418746948], [-0.7018139362335205, -1.4605836868286133, 1.7583853006362915, 2.801577568054199, -0.18837884068489075, 0.2640637159347534, -0.8427716493606567, 0.6832334399223328, -0.16392657160758972, 1.191124439239502, 0.009028336964547634, 1.244480013847351, -0.1737801879644394, 0.03309718519449234, 0.459319531917572, -1.6083264350891113, 0.04278538376092911, 0.3317048251628876, -0.9533424973487854, -0.9128655791282654, 1.321921944618225, 0.05078289657831192, -1.385245442390442, -1.1025261878967285, -0.09428749978542328, -0.05872994288802147, -0.9208178520202637, 0.3483063280582428, -1.1004481315612793, 2.3091955184936523, 0.8913895487785339, -0.2726854979991913, -1.3827131986618042, 1.461344599723816, 0.6404480338096619, 0.701110303401947, -1.46932852268219, -0.3003237843513489, 2.0624659061431885, 0.275162011384964, -0.9516832232475281, 1.0897191762924194, 0.25274497270584106, 1.7349693775177002, 1.6580456495285034, 0.5504106283187866, -1.6359624862670898, 0.4731370806694031, -0.10998530685901642, 0.37377628684043884, 0.762170672416687, -1.2340753078460693, -0.6857361793518066, 0.6912047863006592, -2.2744460105895996, -0.4255867898464203, -0.14751583337783813, -1.7559950351715088, -0.7428008317947388, -0.8027527332305908, 0.2746647000312805, -1.2106342315673828, -0.45446687936782837, -0.2342480570077896, -0.2759159207344055, 0.14532054960727692, 0.4128018319606781, 0.2412729263305664, 2.3108811378479004, 0.13274136185646057, 0.8388292789459229, -0.24110954999923706, -0.5360967516899109, 1.3297905921936035, 0.09771056473255157, -0.4411241114139557, -0.9891069531440735, -0.26431554555892944, 2.1776275634765625, -0.4301277697086334, 0.3738287389278412, -1.0464568138122559, -0.724051296710968, 0.684024453163147, 1.0727680921554565, 0.31950685381889343, -1.604792833328247, -0.9135684370994568, 0.6496855616569519, 0.581537663936615, -0.27635112404823303, -1.023572564125061, 0.7524331212043762, 0.10078735649585724, 0.4646676182746887, -0.8478298187255859, 0.1826411336660385, -0.1250649392604828, 0.8245875835418701, -0.13068132102489471, -0.6154865026473999, -1.3341984748840332, 1.290722370147705, 1.7627904415130615, -2.2225341796875, 0.2843008041381836, 0.261310875415802, 0.008819077163934708, 2.104048490524292, 0.363640695810318, 0.11153515428304672, -0.7135944366455078, -0.2236872911453247, 0.3791678547859192, -1.5474226474761963, -0.08157813549041748, -1.8352525234222412, 0.07096047699451447, 0.43419066071510315, 0.21828573942184448, 0.5639451146125793, 0.9431644082069397, -0.01972009614109993, -1.264461636543274, -0.8137240409851074, 0.28111058473587036, 1.2600250244140625, 0.7804924249649048], [-0.23085889220237732, -0.24503712356090546, -0.0994061827659607, 1.5228581428527832, 0.3767598867416382, -2.3778023719787598, -0.4085182547569275, -0.3943151831626892, -0.25997745990753174, -1.9682143926620483, -1.9871121644973755, 1.530224323272705, -2.530646800994873, 0.391478031873703, -0.5809656977653503, 2.0270884037017822, 0.4882783889770508, -0.8286818861961365, 2.250389337539673, -0.08059035241603851, -0.1935025453567505, 2.502296209335327, -2.409742832183838, 0.7994886636734009, -1.6141458749771118, 0.0005368501879274845, 0.406630277633667, 0.8636829853057861, 0.5089342594146729, 0.8952111005783081, -1.540614366531372, 2.280562162399292, -0.37795358896255493, -1.6547335386276245, -0.15042151510715485, -0.25600069761276245, 0.06788117438554764, -0.06018105149269104, -1.1021264791488647, 1.3758989572525024, 0.6128628849983215, -0.26600757241249084, 0.10612695664167404, -0.30117377638816833, 1.6066874265670776, -1.1868438720703125, 0.5452332496643066, 0.14898936450481415, 0.60993492603302, 1.7433713674545288, 1.0875258445739746, -1.0286593437194824, -0.1958550065755844, 1.4011727571487427, 0.4677949547767639, 0.15467749536037445, 0.22360830008983612, 0.28678134083747864, 0.35054144263267517, 0.886756420135498, -0.48970603942871094, -0.213547483086586, 2.134082794189453, 0.994879424571991, -1.3563746213912964, -0.46263599395751953, 0.8560240864753723, 0.21997298300266266, -1.0666797161102295, -0.823763906955719, 1.0458463430404663, 0.22325411438941956, 1.021500825881958, 0.3973235487937927, 0.06086832284927368, 0.3048955798149109, -0.1894264966249466, -0.9283446669578552, -1.6677782535552979, 0.04926276206970215, 1.8199771642684937, -0.7849177122116089, -0.4952751398086548, 1.3822661638259888, -1.219954252243042, -1.00899338722229, -0.9775038361549377, -0.20179925858974457, -1.022011399269104, -1.3623145818710327, 0.06589219719171524, -0.544605553150177, -0.8883278965950012, 0.12701183557510376, -1.1192346811294556, 0.31269463896751404, 1.8037900924682617, -1.3689745664596558, -0.22134636342525482, 1.5849928855895996, -0.6581693291664124, 1.5323147773742676, -0.1904028207063675, 2.3358752727508545, -1.76125168800354, 1.5519654750823975, 1.1644928455352783, -0.7296068668365479, -0.48667219281196594, 0.01994750089943409, 0.15367722511291504, 0.3024505078792572, -0.23161058127880096, 0.5542299151420593, 1.7278153896331787, 2.649691343307495, 0.5105394124984741, 0.5662274360656738, 0.4985789358615875, -0.7658082842826843, -0.05976259708404541, -0.9302429556846619, -0.6162577867507935, 0.4331553876399994, 0.3210175335407257, -0.8920432925224304, 1.0303906202316284, -1.0400110483169556], [-1.8759139776229858, 2.6851699352264404, 0.13312643766403198, 1.8424122333526611, -0.5943405628204346, 1.7413103580474854, -0.0916721522808075, 1.3762274980545044, 1.5860161781311035, -0.24829082190990448, -0.5742754936218262, -1.0244789123535156, -0.8377402424812317, -1.35868239402771, -0.44602131843566895, -0.7648663520812988, 0.8385955691337585, 0.45884695649147034, 0.1636103093624115, 0.2392435371875763, -0.5982471108436584, 0.7253361344337463, -0.6390175819396973, 0.19464798271656036, 0.1934727132320404, 0.8989622592926025, 0.15333238244056702, 1.0808519124984741, 0.5528156757354736, -1.0137606859207153, -1.9264658689498901, -1.1825491189956665, 1.4036117792129517, -0.2989504337310791, -1.807563304901123, -0.8519845008850098, -0.7940414547920227, 0.0820012241601944, -1.7093884944915771, -0.8938236236572266, 0.83912593126297, -0.48626089096069336, 0.1491188406944275, 0.2231285572052002, -0.14122498035430908, -2.093846082687378, -0.13445714116096497, 0.6492867469787598, -0.03548985347151756, 0.636279821395874, -0.42882856726646423, -1.4162242412567139, -2.1899726390838623, -0.06740934401750565, 1.0697522163391113, 1.2133179903030396, 0.023753762245178223, -0.8793015480041504, -1.467341661453247, -0.6819289326667786, 0.2977820038795471, 0.022828031331300735, 1.0354949235916138, -0.2513805031776428, -0.4253581166267395, 0.4290332496166229, 0.881929337978363, 0.7939648032188416, 0.49770522117614746, 0.5970193147659302, 0.3886907398700714, 0.2570410370826721, 1.130129098892212, 0.4333391785621643, -1.52895987033844, 1.4124659299850464, -1.8366426229476929, 0.33162546157836914, 1.042850375175476, -1.6596561670303345, -1.2004975080490112, -0.3674185872077942, -0.27946680784225464, 1.923265814781189, 0.22605162858963013, -1.9725770950317383, -0.7514937520027161, -1.7424668073654175, 0.9115951061248779, 0.10207908600568771, -1.5578811168670654, 0.8457282781600952, 0.583453893661499, -1.0407453775405884, -1.1563518047332764, 0.5301051139831543, -0.7654587626457214, 0.15143801271915436, 0.5195063352584839, -0.813954770565033, -1.0472925901412964, 0.2183315008878708, 0.4547358453273773, 1.06486177444458, 0.10202697664499283, 0.6251698136329651, -0.9625921249389648, -0.7733641862869263, -1.3567179441452026, 0.5319321751594543, 0.2002263069152832, 0.019120097160339355, 0.6147540211677551, -0.6167978048324585, -1.6529961824417114, -1.4662706851959229, -0.09471850842237473, 0.5981481075286865, 0.9924198985099792, 0.8483686447143555, 0.057439174503088, -1.6388745307922363, -1.2167891263961792, 0.6757595539093018, 2.4529175758361816, 0.48230791091918945, 1.730438232421875, -1.6195247173309326], [0.1266629844903946, 0.7132787704467773, 0.17411907017230988, 0.01648632250726223, 0.7742635607719421, -0.6504238843917847, -1.6602121591567993, -0.12891528010368347, -0.01968221552670002, -0.11526501178741455, 0.0971381813287735, -0.18784040212631226, 0.053611647337675095, -0.2443685531616211, -0.3139667212963104, -0.5497015118598938, 0.5009853839874268, -0.6075757145881653, 1.0399596691131592, -0.037549640983343124, -1.2477827072143555, 0.610662043094635, -0.9163693785667419, -0.5121087431907654, -0.545479416847229, 1.875425100326538, -0.13549906015396118, 0.5738928318023682, 0.5248745679855347, 0.00012240953219588846, 0.2994735836982727, 2.7594668865203857, 0.8235890865325928, 0.6219112873077393, 1.7074058055877686, 1.7457282543182373, -1.0463883876800537, -1.822631597518921, -0.32549408078193665, -1.6054755449295044, 1.0650378465652466, -1.4225913286209106, -0.5784010887145996, -0.559927225112915, -0.3894321322441101, 0.04508696123957634, -1.2451701164245605, -0.43688952922821045, 1.247458815574646, 0.5539882183074951, 0.665587306022644, -1.1515966653823853, -0.11396528035402298, -1.5159779787063599, 1.2358711957931519, -0.826541543006897, 0.4200899600982666, 0.7666065692901611, 0.48862341046333313, -1.7328392267227173, -0.9944868683815002, 0.3922629654407501, 1.2815449237823486, 0.3033628463745117, 0.3182615339756012, -0.8198306560516357, 0.7920544147491455, 0.510837733745575, 0.005407175049185753, 1.6440341472625732, -1.400926113128662, 0.3085535168647766, 0.8407788276672363, 0.44083672761917114, 0.19417721033096313, -1.518492341041565, -0.05332149192690849, -0.6971551775932312, 0.12333624064922333, -0.7568318247795105, -0.6455942988395691, -0.30576780438423157, 0.35553646087646484, 0.29660940170288086, -0.4285580813884735, -1.9633067846298218, -0.6228165030479431, -0.7592499852180481, 0.49171754717826843, 0.4705520570278168, 0.8763507008552551, 1.0626267194747925, -0.0676078349351883, -0.6491996645927429, 0.04286820441484451, 0.8366044759750366, -0.07104671746492386, 0.6667096614837646, -0.19281533360481262, 0.12166933715343475, -0.42244723439216614, -0.7939180135726929, -0.8961280584335327, -0.13867469131946564, -0.8495218753814697, -1.1427854299545288, -0.5309857726097107, 0.3533003032207489, 0.9349058866500854, -0.1797235608100891, -0.6160224080085754, -0.5392829179763794, 1.716121792793274, 0.881328284740448, -0.19718612730503082, 1.1524165868759155, 1.6148940324783325, 0.966292679309845, -1.1054428815841675, -1.1238473653793335, -0.9043038487434387, -2.056361198425293, -0.5502076148986816, -1.4487097263336182, 0.7567209601402283, -0.053238511085510254, -0.5724897384643555, 1.930069088935852], [0.07028297334909439, 0.011017889715731144, -0.44224002957344055, 0.7258009910583496, -0.13401249051094055, 1.5507937669754028, 0.008994041942059994, -1.4545011520385742, -0.1174834594130516, 0.43059977889060974, 1.061769962310791, -1.71010422706604, 0.37355276942253113, 1.198317050933838, 0.8743902444839478, 0.5800586342811584, -0.5941860675811768, -0.9605547189712524, -0.012869109399616718, 0.5656762719154358, 0.3375485837459564, 1.034430742263794, -0.18936172127723694, 0.9988725781440735, -0.2607300877571106, -1.2501760721206665, -1.4274609088897705, -0.45853710174560547, 0.16003172099590302, 1.0435075759887695, 2.4320199489593506, 0.3152453303337097, -0.5115793943405151, -1.8643798828125, -1.7762528657913208, 0.352225661277771, 0.044449783861637115, -0.7772936224937439, -0.5122349858283997, -1.2372130155563354, 0.43650102615356445, 0.3460478186607361, -0.42112109065055847, 0.948893129825592, 0.15834474563598633, 0.2819390296936035, -0.9933247566223145, 0.10035346448421478, -1.6200342178344727, -0.6469377875328064, 0.23628588020801544, -0.5518209934234619, 0.26633384823799133, 0.1977151781320572, 0.880749523639679, 0.16849671304225922, 1.4923056364059448, -0.5943331122398376, -1.291998267173767, -1.5035890340805054, 0.7955819964408875, 0.9787901639938354, 0.25214311480522156, 0.9846441745758057, 2.8310582637786865, 0.35091617703437805, 0.7484582662582397, 0.18604321777820587, 0.735343873500824, -1.3415188789367676, 1.1770797967910767, -0.43860360980033875, -0.5257249474525452, -0.36998450756073, 0.35576239228248596, -1.8761886358261108, -0.9121496081352234, 1.1666278839111328, 1.786571979522705, 0.0499751977622509, -1.2163203954696655, 0.21881331503391266, 1.4372777938842773, -0.004758695140480995, -0.6754593849182129, -0.08940586447715759, -0.09596101194620132, 0.6306297779083252, -1.3872226476669312, 0.11577624082565308, -0.8689897060394287, 1.0200464725494385, -0.5604854226112366, -1.8884071111679077, -0.13826780021190643, 0.21614140272140503, -0.1740795224905014, -0.7415841817855835, 0.7819700241088867, -0.9063917994499207, -1.2261768579483032, -0.5658090710639954, 1.1148878335952759, -0.9239221215248108, 0.6356029510498047, -0.6944913268089294, -0.2876635193824768, 0.769370436668396, -0.2759549021720886, 1.6313883066177368, 0.6239590048789978, 0.15314151346683502, 1.7128758430480957, -0.4699213206768036, 0.24218326807022095, -1.2784067392349243, -0.009089570492506027, 2.3480637073516846, 0.8434836864471436, 0.01620287261903286, 0.9227202534675598, -0.10520044714212418, 0.8553724884986877, -0.22032827138900757, -2.923473358154297, -0.8336485624313354, 2.5066890716552734, -0.21572186052799225]], [[-0.1491931527853012, -2.723600387573242, 0.14034341275691986, 0.18114934861660004, 1.7606216669082642, -0.5308008193969727, 0.8479586243629456, -1.0139573812484741, 1.5881439447402954, -1.2162034511566162, 2.882096767425537, -0.18028731644153595, 0.29493221640586853, -3.0598769187927246, 0.03289276733994484, 0.4587326943874359, 0.6902979016304016, 1.1375126838684082, -0.8335013389587402, 0.3465495705604553, 0.877558171749115, -1.8446505069732666, -1.039265751838684, 0.1708545833826065, 0.5358225703239441, -0.4791911840438843, 0.53892582654953, 0.7486238479614258, 0.5951870679855347, 0.944441556930542, 1.132901668548584, 2.3142640590667725, -1.1915528774261475, -1.0626847743988037, 0.33607128262519836, -1.3994371891021729, -1.146924614906311, 0.8048018217086792, 0.13200165331363678, 2.0092084407806396, -0.21087487041950226, -0.4611562192440033, -0.9634709358215332, -0.6146712899208069, 0.017683131620287895, 1.7226201295852661, 0.25763943791389465, 1.0960122346878052, -1.6601004600524902, -0.048776380717754364, 0.10418334603309631, -0.36020031571388245, -0.6416012048721313, -0.9507429599761963, 1.7964727878570557, -1.5175108909606934, -0.37358230352401733, -0.4827733337879181, -1.6692659854888916, -0.5057381391525269, -0.3517686426639557, -0.6814179420471191, -1.2528553009033203, 0.26977139711380005, -0.08602141588926315, 1.3966869115829468, -0.4543505907058716, 0.1424078494310379, -0.44803816080093384, -1.714307188987732, 2.4356439113616943, 0.5660772919654846, 1.3915112018585205, -0.4553997814655304, 0.28015950322151184, -1.0505927801132202, -2.210597276687622, -0.5901999473571777, 0.3253386914730072, -1.0425047874450684, 0.8195679783821106, 0.8948797583580017, 0.17919358611106873, -0.49569177627563477, 0.9129974246025085, 0.2913285195827484, 1.6700584888458252, 0.15483029186725616, -0.4093132019042969, -1.1008104085922241, 0.7255944013595581, -0.18117652833461761, -1.305570125579834, 1.9027422666549683, 0.06335370242595673, 1.1781631708145142, 0.054591983556747437, -0.17086534202098846, 0.31136971712112427, 0.404520183801651, 0.2331288903951645, -0.6206125020980835, 0.035111092031002045, 0.11127255111932755, 0.6458486318588257, -0.4972419738769531, 1.1666364669799805, -1.33799409866333, 0.5181519985198975, 0.15184007585048676, 0.25314104557037354, -0.567825198173523, -0.3817930817604065, -0.23486855626106262, 0.3818621337413788, -0.5793585181236267, -0.5257527828216553, -0.18238328397274017, 0.20067480206489563, -0.14919322729110718, -0.6140449047088623, 0.9209868311882019, -0.3388573229312897, 1.1817249059677124, -0.4926671087741852, 1.0341531038284302, 0.5691871047019958, -0.5032326579093933], [1.0654882192611694, -0.44520384073257446, 0.0011861595558002591, 1.1769285202026367, 2.785506248474121, -0.1229119524359703, 0.0065079801715910435, 0.07061892002820969, -0.2617415189743042, -0.23107650876045227, -1.3303688764572144, -0.2991085350513458, -0.9250990748405457, -0.15777063369750977, -1.120251178741455, 1.2409451007843018, -0.027401011437177658, -0.871234655380249, 0.89914470911026, 1.6458169221878052, 0.6915668845176697, 0.7577195763587952, 1.7911925315856934, -0.3634980320930481, 0.5290787816047668, 0.620533287525177, -1.1686958074569702, -1.0805482864379883, -0.49485909938812256, 0.6755532622337341, 0.44843780994415283, 1.6067866086959839, -2.010824680328369, -0.8456065058708191, -1.2184921503067017, -0.20579245686531067, -0.10530033707618713, -0.3010634183883667, -0.8434767127037048, 1.121380090713501, -0.17773054540157318, 1.065850019454956, -0.6105843782424927, -0.29652756452560425, -0.9940998554229736, 0.21327504515647888, -0.023786524310708046, 0.5983050465583801, -1.3486077785491943, -0.552651047706604, 0.9062466025352478, -0.5430557131767273, -0.3292710781097412, -0.5269417762756348, 1.5648040771484375, -1.2963292598724365, -0.21907442808151245, 0.6438120007514954, -0.3711644113063812, -0.56185382604599, -1.2610228061676025, -0.9439774751663208, 0.9657939076423645, 0.9168999195098877, 1.5755435228347778, 0.9695945382118225, -0.23317895829677582, 0.12960000336170197, -0.8823639750480652, -1.078245997428894, 0.4230751693248749, -1.070491075515747, -0.031249143183231354, -1.7172526121139526, -0.6143820285797119, 0.8488307595252991, 0.09538538008928299, 0.19266961514949799, 1.3311537504196167, -0.5830046534538269, -0.3545113503932953, -0.2871295213699341, 1.475373387336731, -0.7606687545776367, 0.7584764957427979, 0.838858962059021, -1.120898962020874, 1.9431499242782593, 0.2268611639738083, -0.68062424659729, -0.2878553867340088, 0.08027898520231247, -1.8492748737335205, -0.3920350670814514, -0.17018792033195496, -1.4266936779022217, 0.44649943709373474, -0.28239089250564575, 0.4167342782020569, 0.9248862266540527, -0.1938474178314209, 0.4589042663574219, 0.5191860198974609, 0.08029969781637192, -1.5857274532318115, -1.5514919757843018, -1.3086142539978027, -0.6533926725387573, -1.6427401304244995, 0.5511133074760437, 0.7214766144752502, 0.6691862344741821, 1.1509320735931396, -1.6511949300765991, 0.9302816390991211, 0.9132335782051086, -1.0175501108169556, 0.17524904012680054, 0.6299987435340881, 0.3149893283843994, 1.1962119340896606, 0.36948564648628235, 0.4102129638195038, -1.3729591369628906, 1.8870669603347778, 0.6650540232658386, 0.805519700050354, 0.16434377431869507], [0.6244564056396484, 0.560002326965332, -2.2022039890289307, -0.24513384699821472, -1.4731383323669434, -0.6995742321014404, -0.224625825881958, 1.119187831878662, 0.7933093905448914, -0.3379330635070801, 0.7994054555892944, 1.9750354290008545, 0.8283147811889648, 0.7641223669052124, 0.5292574167251587, 2.5677645206451416, -0.022789902985095978, -1.066304326057434, 0.21571071445941925, -0.46862056851387024, -0.725099503993988, -0.7919829487800598, 0.6423741579055786, 1.460508942604065, -1.409122109413147, 1.132407307624817, 1.6261770725250244, -0.005765100475400686, 0.12178844213485718, -0.603282630443573, -0.7550631761550903, 1.0657379627227783, -0.032202210277318954, 0.9620018005371094, -1.5431263446807861, -2.0706586837768555, -0.035156041383743286, -0.06494613736867905, -0.0018692993326112628, -0.2587210536003113, 0.14294423162937164, -0.788449764251709, -0.8195188641548157, -1.0870765447616577, 0.3094947934150696, -0.5828223824501038, -1.1986521482467651, -1.345719814300537, -0.15968577563762665, 0.4294653534889221, 0.2411523312330246, -2.307791233062744, 0.7661487460136414, -0.6677924394607544, 2.2172884941101074, 0.18402788043022156, 0.6182763576507568, 1.0606298446655273, -0.030613694339990616, -1.6721147298812866, 0.47470560669898987, -0.02943974733352661, -0.9642027616500854, -0.3352106213569641, 0.8657103180885315, -2.6445257663726807, -0.4160696268081665, 0.31366321444511414, -0.315140962600708, 2.17425537109375, 1.5642625093460083, 2.617558240890503, 1.2729685306549072, -0.46600615978240967, -1.8876465559005737, -0.8891564607620239, -0.5391653776168823, -2.6701619625091553, 0.8359047770500183, 0.9598001837730408, 1.1401937007904053, 0.12995202839374542, 0.9509565830230713, 1.2627822160720825, -0.300772100687027, 2.069411039352417, 1.4049549102783203, 1.574939250946045, 0.20559793710708618, 1.2792768478393555, -0.06699288636445999, 0.3112342059612274, -0.411065012216568, 0.7289571762084961, -2.560270309448242, -1.2262202501296997, 0.42907804250717163, -1.302071452140808, 0.8218460083007812, -1.4063720703125, 1.1623364686965942, 0.48093441128730774, 0.48425769805908203, -0.5118521451950073, -0.8339831233024597, 0.7264813780784607, 0.35557234287261963, -0.5450859069824219, -0.8375826478004456, -0.3305600881576538, 1.3184353113174438, -0.05687379837036133, 2.165654420852661, 1.183701992034912, -1.3302381038665771, -0.7837039828300476, -0.591200053691864, 0.5204764604568481, -0.6137369871139526, 0.7476462721824646, 0.6998095512390137, 1.1242262125015259, 1.3159741163253784, 0.6259714961051941, 0.26228243112564087, 0.04270249977707863, -0.3323564827442169, -0.04568769410252571], [-0.1255260407924652, -0.2483312338590622, 1.0515190362930298, 0.30665701627731323, -0.7504348158836365, 0.15989626944065094, 0.29436591267585754, -0.040989942848682404, 1.1484614610671997, -0.25159963965415955, 0.11644653230905533, -0.42617037892341614, -0.6128898859024048, 0.18631891906261444, -0.0033173016272485256, -1.4641889333724976, 0.9662259221076965, -0.20621876418590546, -1.2484374046325684, -0.2320847362279892, -0.14241090416908264, -0.6172454953193665, -1.5511690378189087, -0.6823927164077759, 0.26506859064102173, -1.2195292711257935, 1.480137825012207, 0.6927444934844971, 1.0971006155014038, -0.7347821593284607, -0.5812551379203796, 0.2287779599428177, 0.4589880108833313, -0.5480796694755554, 0.42291852831840515, 1.6604241132736206, -0.817826509475708, -0.14077430963516235, -0.003850211389362812, -1.3979542255401611, -0.41608908772468567, 0.08570689707994461, 2.011105537414551, 1.4702198505401611, 0.6467748880386353, 0.06713468581438065, 0.19094271957874298, -0.13055314123630524, -0.21996815502643585, 0.2610361576080322, -1.6768454313278198, 0.3234778046607971, 0.8753862380981445, -0.3622390925884247, -0.25807496905326843, -2.3690905570983887, 0.3951031565666199, 1.3790465593338013, 1.828722357749939, -0.23590172827243805, 0.5777588486671448, 0.7060319185256958, -0.010373040102422237, 1.971893072128296, 1.0610677003860474, 1.0367388725280762, -0.24303865432739258, -0.04143710806965828, 1.030773401260376, 0.043246764689683914, -2.233856201171875, -0.07244550436735153, -0.07740101218223572, 0.6264290809631348, -1.209573745727539, 0.14496634900569916, -1.4591885805130005, -1.8418083190917969, -2.205538749694824, 1.4238362312316895, 0.8544243574142456, 2.1741814613342285, -0.25424420833587646, 1.0984214544296265, 1.4515653848648071, 1.591270923614502, 1.340582251548767, -0.6076294779777527, 2.4030604362487793, -0.02001769281923771, -0.8169773817062378, 0.5152010321617126, 0.7521247863769531, -0.715155303478241, 0.6450141668319702, -0.6875077486038208, 1.157793641090393, -0.24458415806293488, -1.1169780492782593, 0.5237895250320435, 0.5561850070953369, -1.6868548393249512, -0.9428256154060364, -1.0531623363494873, -0.9743099212646484, 0.3600311577320099, -2.106962203979492, -0.08816505968570709, -0.4111151695251465, -0.7132019996643066, -0.23697875440120697, 0.8002992272377014, -0.6706944704055786, 1.1334106922149658, -2.039602041244507, 2.0688021183013916, 0.776216447353363, -0.336617112159729, -2.831420660018921, 0.8142120838165283, 2.159376382827759, 0.2875169515609741, -1.949579119682312, -1.0098381042480469, -0.866178035736084, 0.2298029661178589, 1.4219754934310913, 0.7212263941764832], [-1.4425679445266724, 0.7757944464683533, -0.417195588350296, -0.07100087404251099, -0.854896605014801, 0.2976458966732025, -0.45304715633392334, -0.22644631564617157, 1.101658582687378, 0.77008056640625, 1.0127297639846802, 0.3571564555168152, 0.7934132814407349, -0.43289652466773987, 0.8832012414932251, 0.288896769285202, -0.5904221534729004, 0.8416600823402405, -1.462851881980896, -1.7110317945480347, -0.07380323112010956, -1.1332018375396729, 0.8919666409492493, -0.05936955660581589, -1.382189393043518, 0.2806369364261627, 1.846583366394043, -1.1443297863006592, -0.2729923725128174, 0.9644069075584412, -1.878820538520813, 0.2598552405834198, -1.4276803731918335, 0.6769492626190186, -0.8564165830612183, 0.10462016612291336, -0.01858106441795826, 0.1317707598209381, 1.4878203868865967, -1.5130864381790161, 1.0785534381866455, 0.7492492198944092, -1.317929744720459, 1.3001434803009033, -0.20113423466682434, 0.7837710380554199, -0.1278039664030075, -1.7692621946334839, 1.622787356376648, 0.8016353845596313, 2.2330572605133057, -1.3006595373153687, -0.35462871193885803, 0.008746262639760971, -1.439471960067749, 1.1452094316482544, 0.8829689621925354, -0.001926289638504386, -0.007285784929990768, 0.21156306564807892, 0.6961507797241211, 2.083916664123535, -0.35919189453125, -0.3701980710029602, -1.479514241218567, 0.9718222618103027, 1.2460229396820068, 1.2758958339691162, -1.7062417268753052, -1.3061435222625732, -0.2184477001428604, -0.628551721572876, 0.03134969621896744, -0.09140412509441376, -2.3734984397888184, 0.852217972278595, 0.8915342092514038, 0.7650887370109558, -1.5763105154037476, -2.3779566287994385, -0.5214527249336243, -1.1581324338912964, 0.5193041563034058, 1.5732495784759521, -0.9714948534965515, -1.0092220306396484, 1.2016617059707642, 0.505903959274292, 0.3815787434577942, -0.9095214605331421, -1.1501786708831787, -1.4538097381591797, 0.5964634418487549, -0.9403727054595947, -0.47260671854019165, 2.6477208137512207, 1.235751986503601, 0.2103758603334427, -0.38540881872177124, 1.0932176113128662, -0.5688467025756836, 0.10018224269151688, -1.2657709121704102, 0.9808087944984436, -0.42778921127319336, 0.8850533366203308, 0.4634717106819153, -1.352202296257019, -0.008200489915907383, -0.43892717361450195, -1.3438587188720703, 0.5099247694015503, 1.3740994930267334, -0.40282177925109863, 1.6395204067230225, 0.5136550068855286, 1.635562777519226, -1.44670832157135, 0.7407342195510864, -1.2789559364318848, 0.22640538215637207, 1.2403477430343628, -1.6582342386245728, 2.7903597354888916, -1.0364954471588135, 0.6981875896453857, -0.4375402331352234, 0.037548717111349106], [-0.4057719111442566, -0.8082281947135925, -0.13344405591487885, -1.6092766523361206, 1.8550947904586792, 0.20921453833580017, 0.7471847534179688, -0.41525042057037354, 0.5856802463531494, 0.7082648873329163, 1.7673028707504272, 1.043745994567871, 0.6324877142906189, 0.5757213830947876, 1.7430164813995361, -0.556189239025116, -0.39156267046928406, 0.8546910881996155, 0.18126514554023743, 0.15955105423927307, -0.9871118068695068, -0.42420125007629395, 2.1902337074279785, -0.9748014211654663, 0.19337807595729828, 1.04241144657135, 0.2378588169813156, 1.124860167503357, -0.790926456451416, 0.6464040875434875, -0.7224088907241821, 1.111832857131958, -0.013118946924805641, -0.3525055944919586, -2.0315444469451904, -0.27521148324012756, -1.1261287927627563, -0.8167089819908142, 0.23992589116096497, 0.14725682139396667, 2.0116896629333496, -0.3418722450733185, 1.340532660484314, -1.564734697341919, -0.5331697463989258, -0.6193276643753052, -0.20296171307563782, -0.5894034504890442, 1.7280150651931763, 0.15950602293014526, -2.0473387241363525, -0.42086291313171387, -1.4669671058654785, -0.5630830526351929, -1.597876787185669, 0.7868784070014954, -0.31251829862594604, -0.3519337773323059, -0.17609445750713348, 1.350873351097107, -0.3516387641429901, -0.06223282963037491, -0.15591083467006683, 0.5812435150146484, 2.0610926151275635, 0.939587414264679, 0.8248903155326843, 0.05416315421462059, -1.1000487804412842, 0.5981544256210327, 1.6872600317001343, 0.691551148891449, 0.3698384463787079, -0.6919203996658325, -0.4352092444896698, 0.7891544699668884, 0.39715641736984253, 0.567144513130188, 0.43577608466148376, -0.3403491675853729, -0.6956681609153748, 0.2807314097881317, -0.7600475549697876, 1.1195777654647827, -1.4576953649520874, 0.4408356845378876, 0.779253363609314, 2.5271999835968018, 0.14021144807338715, 0.613133430480957, -1.389536738395691, -1.3465900421142578, -0.40463462471961975, -0.75676029920578, 0.551449716091156, 0.509145200252533, -2.1085164546966553, -0.05150644853711128, -0.6825059652328491, 2.9419314861297607, -3.421210289001465, -0.13083693385124207, 1.153159260749817, 1.640886664390564, -0.7097157835960388, 0.08443823456764221, -1.568074107170105, -0.5057312250137329, -1.5770151615142822, 1.000779390335083, 1.1632485389709473, -0.3067438006401062, -1.123037576675415, -0.38526326417922974, 0.03828701004385948, 0.37230828404426575, -1.2455799579620361, -0.9094690680503845, -0.5941553711891174, -0.07621356844902039, 1.1233224868774414, -0.5877768397331238, 1.2292243242263794, -1.5417280197143555, 0.3656196892261505, -1.310785174369812, -0.18908697366714478, 0.2903768718242645], [-1.18159019947052, -2.853602409362793, -0.7469052076339722, 1.4311574697494507, 1.3751434087753296, -0.47270187735557556, 1.2350937128067017, 0.6534584164619446, -1.1128590106964111, -0.741433322429657, 0.7043090462684631, 2.200263261795044, 0.5501486659049988, -1.2862801551818848, -1.319115161895752, 0.7839195132255554, -0.0032796263694763184, 0.16759634017944336, -0.8511933088302612, -0.3909450173377991, -0.49161428213119507, -0.7750004529953003, 0.23747345805168152, 0.5725363492965698, -0.9909383058547974, 0.426665723323822, -1.6215015649795532, 2.6740176677703857, 0.17081736028194427, -1.0643584728240967, 1.0306731462478638, 0.5720414519309998, -0.6967483758926392, 1.9717227220535278, 0.2796141803264618, 1.2908928394317627, 1.5685555934906006, 0.9822209477424622, -0.12485779821872711, 0.3763085603713989, -2.182077169418335, 0.3206270933151245, 0.3940635025501251, -1.5907578468322754, -1.7852169275283813, -0.14625482261180878, -1.329240083694458, -1.3394041061401367, 0.49200475215911865, 0.231955423951149, -1.1079797744750977, 0.6760997176170349, 0.09779257327318192, 0.697921097278595, 0.22400449216365814, 1.4185093641281128, 0.3343333303928375, 0.3704836070537567, 1.1858121156692505, -0.31541967391967773, -0.11195559799671173, 1.3093900680541992, 0.6644707918167114, 1.383278250694275, 0.26103368401527405, 1.5897512435913086, 0.14284385740756989, -0.02681814320385456, 1.5639448165893555, 0.27597373723983765, -0.2300683557987213, 0.03348264843225479, 0.0074106487445533276, 3.0078070163726807, -0.8754600882530212, 0.29231688380241394, 1.0271838903427124, -0.4050833284854889, 0.06998774409294128, -0.12271136045455933, -0.3770434558391571, 2.6161675453186035, 0.10556880384683609, -0.06141689792275429, -0.5654107928276062, 2.141411066055298, 0.04263293370604515, 1.181694746017456, 1.6473586559295654, -1.4971590042114258, 0.17493829131126404, 0.8048452734947205, -0.15181086957454681, 0.50437331199646, -1.1282048225402832, -0.6175025105476379, 0.08997227996587753, 1.4144999980926514, 1.4885555505752563, 1.344636082649231, -0.5567273497581482, -0.40410715341567993, -0.043684590607881546, -0.2395247519016266, 0.3576992452144623, -0.6612774133682251, -0.8724850416183472, 0.18931902945041656, 0.6508265733718872, -1.5406795740127563, -0.4064224362373352, -0.48002713918685913, -0.14644566178321838, -0.3633583188056946, -0.8733577132225037, 1.2775671482086182, -0.5274270176887512, -0.6653589606285095, 0.3303438425064087, 0.32175156474113464, -0.019399750977754593, -1.1991163492202759, 0.6504170894622803, -0.6171306371688843, -0.41756951808929443, -0.1159808337688446, -0.9480528235435486, -0.37844595313072205], [1.4261125326156616, 1.7454417943954468, -1.3706740140914917, 1.5155543088912964, 0.635966956615448, 0.7158016562461853, -0.25127875804901123, 0.20131926238536835, -0.3890705704689026, 1.6741679906845093, -0.43127334117889404, 0.6550419330596924, -1.9013344049453735, 0.9359464645385742, -1.0078434944152832, 0.014354084618389606, 0.4329327344894409, -0.43334320187568665, 0.40013617277145386, 0.25210633873939514, 1.4439613819122314, -1.259641170501709, 0.03672923520207405, 0.573117733001709, -0.0006891794037073851, 0.14829465746879578, -0.5015779733657837, -1.9932703971862793, 0.09285709261894226, -1.1500924825668335, -1.601558804512024, 1.602212905883789, 0.33732160925865173, -1.065605640411377, -0.5795722603797913, -0.880671501159668, 0.7282811999320984, 0.7335541844367981, -0.7726243734359741, -0.5672714710235596, -0.014007321558892727, 0.12230179458856583, 1.0609550476074219, -0.814302384853363, -0.07995474338531494, -1.0532796382904053, -0.6502231955528259, 0.004274043720215559, 1.263611078262329, 1.568054437637329, -0.8298692107200623, -1.1822125911712646, -0.33404481410980225, -0.3797668218612671, -0.596289336681366, 0.42519083619117737, -0.29354774951934814, 0.0051935999654233456, 0.6367382407188416, -1.8227968215942383, 0.7180628180503845, 0.6833981275558472, -0.3774120509624481, 0.6127173900604248, -2.8403656482696533, -1.0432065725326538, 0.10963939130306244, -0.5921295285224915, 0.28366976976394653, 0.9138676524162292, -0.026041043922305107, 0.45972132682800293, 0.032103728502988815, 1.6865900754928589, -0.32070276141166687, 0.2930689752101898, 1.0042099952697754, 1.6751046180725098, -0.8610281348228455, -0.6464002132415771, 1.552473545074463, 0.5603724718093872, 0.2110278159379959, -0.022145569324493408, -0.32350215315818787, -0.028585245832800865, 1.3418911695480347, -0.46677613258361816, -1.3501763343811035, -1.633386492729187, 0.09407810866832733, 0.49825456738471985, 0.07354868203401566, -0.32549065351486206, 0.3237414062023163, 0.29162219166755676, 0.6569802165031433, -0.4019438326358795, 1.3270310163497925, 0.7156696319580078, 0.15888963639736176, 0.12123317271471024, 0.5689932703971863, 0.951763927936554, 0.22079932689666748, -1.0509759187698364, -0.08782978355884552, -0.3133668899536133, -0.508351743221283, -1.179664134979248, 1.2750333547592163, -0.09595831483602524, 1.4847925901412964, 0.384054571390152, -0.1291496902704239, 0.3346727788448334, -0.6440312266349792, -0.3838837146759033, -1.1225106716156006, 1.0677845478057861, -0.7063047289848328, -0.8070732355117798, 0.6552684307098389, -1.3897426128387451, -1.3195934295654297, -0.8827810287475586, 0.18330204486846924, 0.1441407948732376], [-1.203841209411621, -0.9615594744682312, -0.6520026326179504, 0.12830129265785217, -0.5826858878135681, -1.3538119792938232, -0.9205659627914429, 0.16511651873588562, -0.7260349988937378, -1.5344067811965942, 0.6906364560127258, 1.632609248161316, -0.19939085841178894, -1.5809816122055054, 0.44452205300331116, -0.4022102355957031, -0.3680587112903595, 0.20300306379795074, 1.2211878299713135, 0.48073503375053406, 1.9718506336212158, -0.09605864435434341, 0.726855993270874, -1.3680474758148193, -0.5366902351379395, 0.3796271085739136, 0.8577110767364502, -0.26875782012939453, -0.9458696842193604, -0.3026667833328247, 0.6980898380279541, 0.9698288440704346, 0.20232130587100983, 1.2616503238677979, 0.25328153371810913, 0.3911353647708893, -0.8644524216651917, 0.5762299299240112, -0.14134946465492249, -1.070089340209961, -0.27012738585472107, -0.3268513083457947, -0.17681176960468292, -1.708709716796875, 1.7895526885986328, 0.7037668228149414, 0.2817249894142151, -0.9021976590156555, -0.22605715692043304, -0.6629077196121216, -0.10785244405269623, 1.5997450351715088, -1.8233445882797241, -1.9722942113876343, -2.139370918273926, -0.5719270706176758, -0.5440998673439026, -1.0300573110580444, 2.0113234519958496, 0.8821192979812622, 1.0374627113342285, -0.7211188077926636, -0.7405231595039368, -0.6864811778068542, 0.13203057646751404, 1.896606206893921, -1.1849106550216675, 0.589999794960022, 0.5086396336555481, -0.30426788330078125, 1.0883243083953857, -0.17131631076335907, -1.8712610006332397, 0.20570428669452667, 0.10716738551855087, -1.5699743032455444, -0.04036334529519081, -0.05317481979727745, 0.37893983721733093, 0.24774061143398285, -0.6564105749130249, 1.1458017826080322, -0.9895727634429932, 0.9085968732833862, 0.9054454565048218, -0.6189131140708923, -0.34205862879753113, 0.6005292534828186, 0.08400599658489227, 0.22901420295238495, 0.24541893601417542, -1.316290259361267, -0.10791214555501938, -0.3038225471973419, -1.7265262603759766, 0.13718052208423615, -0.6781437397003174, -0.8964824080467224, -0.43816056847572327, -1.6588343381881714, 1.6431218385696411, -0.32749852538108826, -0.9202348589897156, -0.9089018106460571, -1.5852570533752441, 0.8050102591514587, 0.9570397734642029, 0.41427695751190186, -0.6250059604644775, -0.22187085449695587, -0.01728265918791294, 0.26948443055152893, 2.726728677749634, -1.6671792268753052, -1.197877287864685, 0.21803805232048035, 0.40162357687950134, -0.017259836196899414, 2.603367328643799, 1.1076221466064453, 0.2143556922674179, 1.5811227560043335, -0.08225196599960327, 0.6511576771736145, 1.0923787355422974, -2.089618444442749, 0.916467547416687, 1.3643500804901123], [-0.5583478212356567, -0.3764766454696655, -0.1831696331501007, -0.3577837646007538, 0.4403630793094635, -0.8514299392700195, 0.001309512066654861, -0.6292585730552673, 0.21710555255413055, 1.3992221355438232, -0.7410146594047546, -0.3940129280090332, -0.044207170605659485, -2.101999521255493, -1.396895170211792, 0.5475876927375793, 1.0296659469604492, 0.3880074620246887, 0.2876153886318207, 2.212477922439575, -1.1821690797805786, 0.05335907265543938, -0.09397716820240021, 1.1250827312469482, 0.27764061093330383, -2.7595434188842773, 1.5308527946472168, -0.3816656470298767, 0.026009157299995422, -0.7404240965843201, 0.7607239484786987, 2.433812379837036, -0.7062923312187195, 0.5923939347267151, -0.5901599526405334, -1.395585298538208, 0.23111651837825775, -0.6253597140312195, 0.12790223956108093, 1.093032717704773, 1.547109603881836, -0.693370521068573, -0.3690224885940552, -0.3619321882724762, 0.8285886645317078, 0.4915666878223419, -0.18554061651229858, 1.2533129453659058, 0.3521879017353058, -1.363993525505066, 1.0354013442993164, 0.4815558195114136, -1.6516127586364746, 0.04276955872774124, 0.5177088975906372, 1.8666362762451172, -2.309577226638794, 0.23418086767196655, -0.29072174429893494, -0.8166062831878662, 0.7387089729309082, -0.4307343363761902, -0.5643793344497681, -1.735150933265686, -0.9742665886878967, -0.5235045552253723, -0.19482861459255219, 0.2046196311712265, -0.9475417137145996, -0.2118023931980133, -0.23139801621437073, 1.658279538154602, 0.34089547395706177, -0.13473089039325714, 0.8574475646018982, 1.9771337509155273, -1.3221771717071533, 0.8088873028755188, 2.511798620223999, -1.457708716392517, 0.9243426322937012, 1.6144382953643799, 0.6774885058403015, 1.405737280845642, 1.1926429271697998, 1.3819764852523804, 0.4908897578716278, -0.10357638448476791, -1.8773524761199951, 0.33213499188423157, -0.12185655534267426, 0.584196150302887, 1.2937569618225098, 0.16976918280124664, -1.94513738155365, 1.0512536764144897, -0.860837996006012, -0.8926453590393066, -0.5787778496742249, -0.8830888867378235, -0.1585681140422821, -0.9754261374473572, -0.7235100269317627, -0.49786850810050964, -0.17847499251365662, 0.08516241610050201, 0.829940140247345, 0.15474361181259155, -1.2833857536315918, -0.6693815588951111, 0.11148952692747116, 0.171963632106781, 2.6786324977874756, -0.6052752733230591, -1.0081359148025513, -0.004223305266350508, 0.5582552552223206, 0.024769563227891922, 0.3973788619041443, -0.8664761781692505, 1.0469354391098022, 0.8713521361351013, 1.203743577003479, -1.7871952056884766, 0.5148875713348389, 1.9511547088623047, -0.5968382358551025, 0.3610773980617523], [-0.9879357814788818, -1.3083101511001587, -0.2181479036808014, 0.8850144743919373, -0.4284989833831787, 1.049343466758728, 1.274891972541809, 0.2500622570514679, -0.9301916360855103, 0.14941835403442383, 2.427678108215332, -0.5155106782913208, -1.3082231283187866, 0.5238274931907654, -0.6603793501853943, 0.9992715120315552, 0.17939646542072296, -0.13470785319805145, 0.2139335572719574, -0.12053913623094559, -0.1946890950202942, -0.8455128073692322, -0.15952177345752716, -1.193195104598999, 0.8304196000099182, -0.02476181462407112, -0.5964991450309753, -0.9639010429382324, 0.8092243075370789, 1.4443893432617188, 0.7360881567001343, 0.4548065960407257, 1.286281943321228, -0.46039122343063354, -0.21197891235351562, 2.2165474891662598, -0.036671433597803116, 0.08009392768144608, -0.607224702835083, -0.2573966979980469, -0.06542014330625534, 0.7875857353210449, -0.30889084935188293, -1.385628581047058, 0.447417289018631, -1.2629728317260742, 3.3014018535614014, -0.9075909852981567, 0.8220082521438599, -2.001657485961914, 0.35868582129478455, 0.8379891514778137, -0.18931053578853607, -0.9499925971031189, -0.6725357174873352, 0.6177958846092224, 0.1648203730583191, -0.6927975416183472, -0.2848108112812042, 0.4894425570964813, -0.33540257811546326, 1.1761181354522705, -1.1992565393447876, 0.015562747605144978, 0.0024328110739588737, -1.007872462272644, -1.7654072046279907, 1.0248448848724365, -0.16240178048610687, -2.6040799617767334, 1.2479538917541504, 0.9859370589256287, 1.4710471630096436, -0.2872752249240875, -1.2780725955963135, 0.05254880338907242, -0.5638726949691772, 1.3107109069824219, -0.21127645671367645, -0.44676002860069275, -0.22779487073421478, 0.3752559721469879, -0.5176226496696472, -0.5959867238998413, -0.09933441132307053, -0.12684518098831177, 0.8050965070724487, 0.3439185619354248, -0.19076073169708252, -0.3966260850429535, 2.3421263694763184, 0.36496663093566895, 0.2508023977279663, 0.2598416209220886, -0.18667197227478027, -1.1583195924758911, 1.6919159889221191, 0.7795714139938354, -0.017880016937851906, 0.20464298129081726, -0.31193971633911133, -1.2890028953552246, 1.4561755657196045, 1.3416035175323486, -0.14845053851604462, -1.1398813724517822, -1.1252280473709106, 0.3992666006088257, -1.7393697500228882, -0.43769440054893494, -1.03638756275177, -2.5752015113830566, 1.7505744695663452, -0.7334032654762268, -1.250651478767395, -0.1746373176574707, -1.6706453561782837, -1.3019239902496338, -0.32894548773765564, -0.030936021357774734, 0.5486543774604797, -1.3997530937194824, 0.13712434470653534, -0.9055563807487488, -1.0639679431915283, 1.5072920322418213, 2.3164217472076416, -0.6829478144645691], [-0.36354517936706543, -0.42636236548423767, 1.8407022953033447, 0.6253520250320435, 1.3734562397003174, -0.49459171295166016, -0.46132993698120117, -0.21587060391902924, -2.3643124103546143, 0.20469418168067932, 1.1940842866897583, -0.7177186012268066, 1.7171127796173096, -0.5898347496986389, -0.4438422918319702, 0.16666606068611145, -0.6832675337791443, 1.469604253768921, -1.2542685270309448, 2.052581787109375, -2.175313711166382, 1.3563904762268066, -0.575132429599762, -0.29354003071784973, -0.27142658829689026, 1.2434206008911133, 0.5584502220153809, -0.5891945958137512, -1.5158050060272217, -1.7555763721466064, -0.9678031802177429, -1.2212163209915161, 1.3664772510528564, -0.7199677228927612, 0.030525555834174156, 1.2591156959533691, 1.5083115100860596, -0.040498752146959305, -0.783320963382721, -1.249678134918213, 0.05415751785039902, 0.6855045557022095, 1.0550671815872192, 0.4740690588951111, 1.0984997749328613, 0.45635902881622314, -1.4818750619888306, -0.5898245573043823, -1.6145223379135132, -0.570220947265625, 1.4990662336349487, -2.4305801391601562, 0.7069942355155945, -0.9565842747688293, -0.980796217918396, -0.008151233196258545, -2.202329397201538, 2.5231730937957764, -0.9635593891143799, -1.0791015625, -0.14640700817108154, 0.28340718150138855, 1.5181083679199219, 1.0196044445037842, 1.7015838623046875, 1.1072578430175781, -0.07190714031457901, 0.4681924283504486, -1.4035643339157104, -0.6419528126716614, -0.09597595036029816, 0.5906816720962524, -0.7247359752655029, -0.27468177676200867, -1.252814531326294, 0.3637644946575165, -0.9848779439926147, 0.0082432571798563, 0.9238511919975281, 0.06722872704267502, -0.14608411490917206, 0.1104661300778389, 1.3562692403793335, 0.23696522414684296, -0.220514178276062, 0.28172093629837036, 0.9030481576919556, 0.015304495580494404, -0.6168174147605896, 1.5790914297103882, 2.1506423950195312, 0.7991593480110168, -0.7343152761459351, -0.8249358534812927, 0.8429369330406189, -1.1601881980895996, -0.8138615489006042, -2.4207773208618164, -0.24461710453033447, -0.6857385635375977, -1.6572471857070923, -1.14093017578125, 1.3797893524169922, 0.17194126546382904, -0.674067497253418, -0.3230170011520386, -0.4006689190864563, 0.18767254054546356, -1.1803182363510132, -1.0635894536972046, 0.2910084128379822, -0.36550161242485046, -0.018168477341532707, 0.3001783490180969, -1.4280441999435425, -0.5500892996788025, -2.2713353633880615, -1.0271416902542114, 2.9893176555633545, -0.8124693036079407, -1.3910958766937256, -1.0395585298538208, -0.2196832150220871, 0.8809557557106018, 0.9259768724441528, -1.4146511554718018, 0.15749870240688324, 0.5067793130874634], [-1.4418742656707764, -1.4326432943344116, -0.2670764625072479, -0.6813977360725403, -1.0599290132522583, -1.4111888408660889, 0.19555750489234924, -0.63880854845047, 0.6340078115463257, 0.31988900899887085, 0.49249956011772156, -1.7299168109893799, 0.20353543758392334, -2.392094850540161, -1.328077793121338, 1.3546788692474365, -0.09046228975057602, 1.343632459640503, -0.16212423145771027, -1.1194167137145996, 0.31108012795448303, 1.3150371313095093, -0.7095207571983337, -1.4277135133743286, -1.0141644477844238, 0.1513597071170807, 0.38753989338874817, -0.6029528975486755, 1.0900745391845703, 0.8058250546455383, -0.3090236783027649, 0.573745846748352, -1.8154715299606323, -0.5915735363960266, 2.2032933235168457, -0.17363330721855164, 0.2464333325624466, 0.4586256444454193, -0.7347885370254517, -0.8233640789985657, -0.6137421131134033, 0.15730608999729156, -0.832500696182251, 1.2060601711273193, 0.280674546957016, 1.3596892356872559, 1.0993980169296265, -0.5766633749008179, 0.7443957328796387, -0.28227144479751587, -2.0726943016052246, 1.326528549194336, 0.26858359575271606, -0.5893363952636719, -1.8921618461608887, -0.7155972123146057, -0.07019870728254318, 0.1692093163728714, 1.3218634128570557, 0.4712490439414978, 1.2968635559082031, 1.7202658653259277, -1.5995090007781982, -0.4820559620857239, -1.0967496633529663, -0.34529614448547363, 0.40322327613830566, 0.24892671406269073, -0.6497138142585754, -2.1264548301696777, -1.4537031650543213, -0.4342498779296875, 0.23191401362419128, 0.46423473954200745, -0.4760695695877075, 1.5961930751800537, -0.6179954409599304, -0.9012351036071777, 0.02312372624874115, 0.25135353207588196, -0.36452966928482056, -1.288495659828186, -0.8358887434005737, 0.9677667617797852, -1.2435574531555176, -1.1420522928237915, 1.8463480472564697, -1.1128134727478027, 0.14165519177913666, -0.3292084038257599, -0.651656448841095, -1.3986705541610718, 1.4625552892684937, 0.1262672245502472, -1.0224132537841797, -1.337506651878357, 0.5388515591621399, 0.8194653987884521, 0.8748072981834412, -0.6184486746788025, 0.43054160475730896, -0.6354140043258667, -0.46711841225624084, -0.07934312522411346, 0.776913583278656, 0.37796667218208313, -1.4350969791412354, 1.7337415218353271, -0.08508981019258499, -0.009492564015090466, -0.020116999745368958, -0.3210873007774353, 1.057018518447876, 1.7522342205047607, -1.3254448175430298, 0.4336818754673004, -1.986126184463501, -0.4454556405544281, 1.9305336475372314, 0.8823354244232178, 0.722338080406189, -0.5359272360801697, 0.9253324866294861, 0.7570504546165466, 0.38544654846191406, -0.019510285928845406, -0.26879259943962097, -0.1642908751964569], [0.31924375891685486, -0.3946318030357361, -0.9567919969558716, -1.0395877361297607, -0.9880447387695312, -1.6156560182571411, -0.9064640998840332, 0.8438103795051575, -0.5145843029022217, -0.663173258304596, 1.4007444381713867, -0.9003262519836426, 0.19393126666545868, -1.1352579593658447, 0.7548238635063171, -0.09069492667913437, 0.5556057095527649, 2.008524179458618, -0.03571777790784836, -0.5675575733184814, -0.24444343149662018, 0.3449516296386719, -0.4074389636516571, 2.0199193954467773, -1.5113414525985718, 1.0237205028533936, 0.3777715563774109, 1.6044200658798218, 0.017113901674747467, -1.59580397605896, 1.020626425743103, 0.07746929675340652, -0.20113617181777954, 0.20690011978149414, 0.5949859619140625, -0.4019688367843628, 0.947669506072998, -0.19861288368701935, -0.3857319951057434, 0.4221150279045105, -1.9390621185302734, 0.02834896184504032, 0.7376951575279236, 0.9925742745399475, -1.4110069274902344, -1.0579365491867065, -0.6043273210525513, -2.011644124984741, 0.09128021448850632, -0.7210568189620972, -0.22811205685138702, -1.1750588417053223, 0.8407009243965149, 0.10068472474813461, 1.3685009479522705, -0.21258297562599182, -0.9442145824432373, 0.5409788489341736, -0.38102665543556213, -2.010963201522827, -2.2878048419952393, -0.3182578980922699, -1.666313886642456, -1.755649209022522, 0.688110888004303, -0.3490797281265259, -0.7367932200431824, -0.9363977909088135, 1.3215056657791138, -0.7766973376274109, -0.5327622294425964, -0.2859059274196625, -0.4329387843608856, -1.8529653549194336, 0.31515395641326904, 0.6929103136062622, 0.2894611358642578, 0.07185903191566467, -1.0312286615371704, -1.359840750694275, 0.7108418941497803, -0.49791577458381653, -1.0090205669403076, -1.033921718597412, -2.3268070220947266, -0.41069766879081726, 0.4723654091358185, 0.9268057942390442, 0.6767187118530273, 1.184770107269287, 0.17491376399993896, 1.4319838285446167, 0.8660476207733154, -0.9345175623893738, -1.5361742973327637, -1.1148462295532227, 0.17665137350559235, -0.3695240914821625, 1.4212934970855713, -0.7927997708320618, -0.19124872982501984, 0.10537541657686234, -0.3586752712726593, -1.1081535816192627, -0.7199501991271973, -2.5773403644561768, 0.48293378949165344, 0.49883389472961426, 0.48206406831741333, -0.2855207920074463, 0.5459101796150208, 0.5032342076301575, 0.6946613788604736, 0.706355094909668, 0.7082801461219788, 0.5585169792175293, 0.5286029577255249, -0.28618761897087097, -0.3913542330265045, -0.8156011700630188, 0.7336044311523438, -1.972311019897461, -0.12003937363624573, 1.0651509761810303, 0.44645562767982483, -0.1815367192029953, -0.1375264823436737, -1.5196876525878906], [1.113394021987915, -0.15668687224388123, 0.8211494088172913, 1.6941863298416138, 0.2699182331562042, -1.2501190900802612, 1.0725575685501099, 0.34230899810791016, -0.08221208304166794, 1.1337276697158813, -0.12221574038267136, -0.5583428144454956, -0.16786673665046692, 0.5755553245544434, -0.5625889301300049, -1.0853300094604492, -0.32619673013687134, 0.6458284258842468, 0.16308575868606567, -1.0985252857208252, 1.5697755813598633, -0.3845612406730652, -0.13569386303424835, -0.8648427128791809, -0.8469094038009644, -0.19829417765140533, -0.8085392117500305, 1.3264920711517334, 2.280278444290161, 0.24368558824062347, 1.5149219036102295, -0.5814591646194458, 1.817274808883667, -1.649616003036499, 1.9727749824523926, 0.5084307193756104, -0.8265900015830994, -1.197031021118164, 0.2965362071990967, -0.8375332355499268, 0.036106690764427185, 1.3941246271133423, 0.833008348941803, -0.603984534740448, 0.45505374670028687, 0.10307557880878448, 0.17007021605968475, 0.9366786479949951, -0.2924586534500122, 0.39724355936050415, 1.0422883033752441, 0.30269187688827515, 1.2838200330734253, -0.3456633687019348, -0.34571248292922974, 0.37172624468803406, -0.15084956586360931, -2.6151392459869385, 0.3018626868724823, -1.780403733253479, -0.06569700688123703, 0.2856350839138031, 0.8990292549133301, -1.372428297996521, 2.108707904815674, 1.0307903289794922, -1.1784839630126953, 0.32584646344184875, 1.2562693357467651, -0.9066652655601501, 1.2816886901855469, 1.683958888053894, -0.3981189429759979, -0.3215307891368866, 0.8496707081794739, -2.0106019973754883, 1.169026494026184, 0.3640243113040924, 1.783408284187317, 0.6481401920318604, -0.6049346923828125, 1.6509751081466675, 0.2982529401779175, -0.029537376016378403, 0.09209351241588593, -1.2478325366973877, 0.5529107451438904, 1.1992294788360596, 1.2891217470169067, -1.6832021474838257, 0.5241919755935669, -0.037807442247867584, -0.40033844113349915, 0.05186890438199043, 1.3736625909805298, 1.046244502067566, 1.5436354875564575, 0.10144168883562088, 2.122527599334717, -0.6312229633331299, 1.1674922704696655, -0.7247506976127625, 1.4111074209213257, 0.4689164459705353, -0.19181346893310547, 0.11025510728359222, 0.9136728644371033, 0.29515260457992554, -0.12838424742221832, -0.4242539405822754, 1.7329871654510498, 0.6641104817390442, 1.1375555992126465, -0.47797465324401855, -0.6729905009269714, 0.09194138646125793, 0.13892774283885956, 0.3016318380832672, -0.6755359768867493, 0.654765248298645, -0.7429016828536987, 0.6440108418464661, 1.086368203163147, 0.6410067081451416, 0.13629092276096344, -1.4388121366500854, 0.05533469468355179, 1.7924786806106567], [0.4652157127857208, -0.06599415093660355, -0.688768208026886, 0.5465496778488159, 0.8441744446754456, 0.08077896386384964, -1.1611618995666504, -0.5700668692588806, -1.3038029670715332, 0.8922000527381897, 1.104666829109192, 1.5965064764022827, 1.2776838541030884, 1.539791226387024, 0.8563342690467834, -1.355079174041748, -0.23497061431407928, -0.5675452351570129, -1.0054084062576294, 0.11823000013828278, 0.034974757581949234, -0.6692190170288086, 0.4654533863067627, 0.3167324960231781, 0.3669201135635376, -0.7195764780044556, 2.305133104324341, 2.10642671585083, 2.5500102043151855, 1.9741917848587036, -3.1721537113189697, -2.2183926105499268, -0.05914114788174629, -0.6215877532958984, -0.8186403512954712, 0.3567880690097809, -0.5721302032470703, -1.6568207740783691, -0.8979910612106323, -0.5857870578765869, 0.3065209686756134, 0.5153859853744507, 1.2232961654663086, -0.7137459516525269, -0.7345934510231018, 0.9566176533699036, 0.7062274813652039, 1.048339605331421, 1.3811328411102295, 0.5637362003326416, -1.8866095542907715, -2.406423330307007, -0.0164746455848217, 0.8210248351097107, -1.144970417022705, -1.2217721939086914, 1.021982192993164, -0.298806369304657, -0.9992858171463013, 0.9655370116233826, -0.19540441036224365, 0.7644659876823425, -0.08339736610651016, 0.16075368225574493, -0.08649852126836777, 1.1160228252410889, 0.5864575505256653, 0.6044512987136841, -0.24430891871452332, 0.31872257590293884, 0.0031518652103841305, 1.1282740831375122, -0.5144751071929932, 0.8557822704315186, 0.8844049572944641, -0.15766045451164246, 0.6214897036552429, -0.07985437661409378, 1.1353596448898315, 1.0571469068527222, 0.12373004108667374, 0.17346420884132385, 0.5338905453681946, 1.207816481590271, 2.9653987884521484, -0.2810854911804199, -1.1806706190109253, -0.32562023401260376, 0.8903042674064636, 1.2158989906311035, -0.2099969983100891, -0.2847254276275635, -0.2726063132286072, 2.317610502243042, -1.327785849571228, 0.7077056169509888, 0.09287688136100769, -0.14576974511146545, -0.0606597438454628, -0.9582285284996033, -0.2975309491157532, -0.03373271971940994, -0.37183231115341187, 0.9573272466659546, 0.2691915035247803, 1.7227216958999634, 0.8241528272628784, -2.1364364624023438, -1.1119420528411865, -1.122300386428833, 0.1811058521270752, 0.006600061897188425, 0.5177435278892517, 2.010155439376831, -0.14043943583965302, -0.1657623052597046, -1.7537933588027954, 0.6791484951972961, -1.363244891166687, -1.116864562034607, -1.1186145544052124, 1.3698186874389648, 1.5373828411102295, -2.2047510147094727, 1.4760023355484009, -0.10217135399580002, 0.5951071381568909, 0.5937663316726685], [-0.9606337547302246, 0.39646634459495544, 1.392099142074585, -0.7164736986160278, -0.9032056927680969, -0.5887901782989502, -0.04700399562716484, -1.7863448858261108, -0.6469555497169495, 0.30994200706481934, -0.7847321629524231, -0.22327542304992676, 2.0995795726776123, -1.8374735116958618, 1.6580389738082886, -0.8874431848526001, -0.5468068718910217, -0.7425311803817749, 0.519568920135498, 0.21740968525409698, 1.0636446475982666, -0.22002586722373962, -0.1503538340330124, 1.1660584211349487, 2.202714443206787, 0.7759332656860352, -1.1638429164886475, -0.5719519257545471, -1.162987232208252, -0.7475504875183105, -0.2822033166885376, 0.28632599115371704, 1.0209133625030518, -1.2113335132598877, 2.012735366821289, -0.5500588417053223, -0.4183080792427063, -1.5193190574645996, 0.07862614095211029, -0.3694436252117157, 0.10379254817962646, -0.8187404870986938, 1.4140783548355103, 0.4453018307685852, 1.519228219985962, 0.6613616943359375, 0.5799298286437988, 0.5122302770614624, 0.8885661959648132, -0.5717283487319946, 0.607002854347229, -1.110190749168396, 0.24919646978378296, -2.0111937522888184, 0.724759042263031, 0.3000882565975189, 1.1465922594070435, 1.1483477354049683, 0.02759232372045517, 0.6176049709320068, 1.6826452016830444, -0.987928032875061, -0.12485846132040024, -1.6627998352050781, 1.2373865842819214, 1.3286138772964478, -0.6363837122917175, -0.055398814380168915, 1.118179202079773, -1.1233816146850586, 1.0782777070999146, -0.22548642754554749, -0.0481816790997982, -0.4306533932685852, 1.1961712837219238, -0.25365883111953735, 1.2954189777374268, 0.04325040429830551, 1.2743549346923828, -0.25777164101600647, 0.41089779138565063, -0.26600944995880127, 1.024741530418396, -0.334945946931839, -0.9809252023696899, 0.11530555784702301, -0.793899416923523, -0.39543622732162476, 0.32456669211387634, 2.731048822402954, 1.368802785873413, 0.6440547108650208, 3.5973026752471924, -0.9658355712890625, 0.05118059366941452, -1.000172734260559, 0.09354628622531891, -0.11604322493076324, -1.1915595531463623, -0.5324242115020752, -0.27034419775009155, -0.10332654416561127, 1.2702082395553589, 0.18938668072223663, 1.4139982461929321, 1.2233904600143433, -1.4945435523986816, 0.17475582659244537, 0.1138380616903305, -1.1523072719573975, 0.8364748358726501, -1.0975873470306396, 0.11223554611206055, 1.1261216402053833, -0.44949907064437866, 1.2141096591949463, 1.8519906997680664, -0.5246186852455139, 0.7254968881607056, 1.3650349378585815, -1.1449872255325317, 0.42835721373558044, -0.8090125322341919, 0.07605069130659103, -0.9335256814956665, 1.1094757318496704, -1.753691554069519, -0.5580357909202576], [-0.44813257455825806, -0.13002561032772064, -1.1796488761901855, 1.7040197849273682, 2.5740292072296143, -0.3628362715244293, -1.4873511791229248, -0.8949307203292847, -0.7389956712722778, 1.1064316034317017, 0.3322887122631073, -0.6022297739982605, -1.701270580291748, 0.027715880423784256, 0.688528835773468, -0.2052670121192932, 0.9411601424217224, 1.0349160432815552, 0.8793054819107056, 2.2431445121765137, -0.4461342990398407, 0.17127352952957153, -1.3146735429763794, 1.1539862155914307, -0.37221473455429077, -0.7472473978996277, -0.15494544804096222, 1.4027085304260254, -1.0427416563034058, -0.20948779582977295, -0.9828336238861084, 1.214803695678711, 0.5736692547798157, -0.27333322167396545, -1.1368331909179688, -0.5789265632629395, 2.464247703552246, -0.308917760848999, 2.039888381958008, -0.18614642322063446, 0.8874130845069885, 3.515399694442749, 0.7185774445533752, 0.085578553378582, -0.058984313160181046, 0.03289240598678589, -0.3834611475467682, -0.10082446783781052, -1.1675816774368286, 0.5324037671089172, -1.2730445861816406, 0.6092194318771362, -0.32662275433540344, 0.1051558181643486, -0.18396931886672974, 0.23672620952129364, 0.12463736534118652, -0.19111353158950806, 0.58539879322052, -0.9438038468360901, -0.007427255157381296, -0.23562327027320862, -0.5941821336746216, -1.3130055665969849, -1.8171398639678955, 0.5282537341117859, -2.1415798664093018, -0.5443681478500366, 0.3460162878036499, -0.693494975566864, 1.661835789680481, 0.20139609277248383, -0.8271805047988892, -0.3655720353126526, -1.1470530033111572, -0.8584761619567871, -1.3845231533050537, -0.5442992448806763, -0.27849501371383667, 0.40864524245262146, -0.04024793952703476, -0.21366046369075775, 0.6807454228401184, 0.2132439911365509, -0.4952373802661896, -1.1748322248458862, -0.2604143023490906, -0.36608949303627014, 1.0432394742965698, -0.6579042077064514, -1.1118767261505127, 0.3203701972961426, -1.4511141777038574, -0.8851726055145264, 0.7608875632286072, 0.8805558681488037, 0.7685452103614807, 0.6313405632972717, 0.1702609807252884, 0.5171118378639221, -1.2729439735412598, -0.4947929084300995, 1.3174289464950562, -1.3955464363098145, -0.949017345905304, -1.4421851634979248, 2.2976508140563965, 0.14177045226097107, 0.47120776772499084, 0.19742408394813538, -0.8379572629928589, 1.1355408430099487, -1.4792535305023193, -0.7485025525093079, 1.1445845365524292, -0.31613215804100037, 0.00010693016520235687, 2.8650193214416504, 0.2305491417646408, 1.086990237236023, 0.6605414152145386, 1.8392454385757446, 0.7274016737937927, 0.529229462146759, 1.5995762348175049, 0.8013862371444702, 0.18821898102760315, 0.4867947995662689], [1.2610589265823364, 1.1040867567062378, 1.3513069152832031, 0.08083212375640869, 0.1430976390838623, -0.5596779584884644, -1.1331017017364502, 0.5892532467842102, -0.45365461707115173, -0.023006431758403778, 0.9832850098609924, 1.7434196472167969, 0.5393661260604858, -0.4461016058921814, -1.1306382417678833, -0.3928239941596985, 0.6514911651611328, 0.19703710079193115, 0.7293038368225098, 0.38075172901153564, -0.048145636916160583, -1.070529818534851, 1.0918147563934326, -1.744274616241455, 1.283645749092102, -0.257683128118515, -1.0528883934020996, -1.534787654876709, 0.9051691293716431, 0.25630709528923035, 0.37279748916625977, -0.9072970747947693, 2.3660645484924316, -1.2130372524261475, 0.4627251625061035, 0.10172715038061142, 1.145349383354187, 0.522473156452179, -0.4871786832809448, 0.4381132423877716, -1.8743778467178345, 0.6644921898841858, -0.2134784758090973, 1.6267322301864624, 0.14014174044132233, 1.0808521509170532, 0.23165346682071686, 0.3282281458377838, 0.008684013038873672, 0.21194282174110413, -0.8773210644721985, 0.2684978246688843, -0.7715852856636047, 1.6094129085540771, 0.653928816318512, 0.08655824512243271, 0.6137881875038147, -1.404618740081787, 0.8346943855285645, -0.04535093531012535, 0.16204635798931122, -0.10188685357570648, 0.8363777995109558, -0.8519189953804016, -0.9213001728057861, 0.1358787715435028, 0.26491159200668335, 0.0383525975048542, -0.33404552936553955, -1.5800628662109375, -0.3924385905265808, -0.6847104430198669, 0.39156508445739746, -0.24923986196517944, -0.4707172214984894, -1.1893680095672607, -0.013300882652401924, -0.44122442603111267, -0.8600959777832031, -0.6736850738525391, 1.2481424808502197, 2.067075252532959, -1.1450635194778442, -0.6233381628990173, -0.7492883801460266, -1.4971725940704346, -0.6009112000465393, 1.6995463371276855, 1.9244846105575562, -0.5819910764694214, 0.5569407939910889, 0.31877610087394714, 1.0952353477478027, -1.500611662864685, -0.14794135093688965, -0.6186646819114685, 0.023701459169387817, 0.252429723739624, -0.7948124408721924, 1.6323732137680054, -0.40749862790107727, 0.5492081642150879, 1.7750481367111206, -1.0299443006515503, -0.09650078415870667, 0.12138460576534271, 0.4905315637588501, 1.6734687089920044, 0.209230437874794, 0.9455612301826477, 0.7263182401657104, -0.49730074405670166, -1.393373966217041, -0.8358367681503296, -0.8541744351387024, 0.5686931610107422, -0.308051735162735, -0.3047560453414917, 1.2013745307922363, -0.027404403313994408, 0.025461668148636818, -0.5451793670654297, 1.377231240272522, 2.2893195152282715, 0.6166407465934753, 0.24158617854118347, -1.3082209825515747, -0.14342433214187622], [0.0886760726571083, 0.14445605874061584, -0.5173537731170654, 0.7088020443916321, 0.9305999279022217, 0.16152283549308777, 1.1037280559539795, 1.12157142162323, -0.589370608329773, 1.446583867073059, -1.8022855520248413, 0.12504161894321442, -1.005899429321289, -0.9875966906547546, -0.5088398456573486, -0.08237573504447937, -0.18672417104244232, 0.3970356583595276, -2.3970091342926025, 0.7484102249145508, 1.917128562927246, -0.71409672498703, 0.9541494250297546, -0.9995611906051636, 0.04461228474974632, -1.5975565910339355, 0.4307488799095154, -0.030641799792647362, 1.6966687440872192, -0.6294474005699158, 0.8802637457847595, 0.054764099419116974, 0.5182934403419495, -1.0857545137405396, 0.30837079882621765, 1.0177428722381592, -0.2797420620918274, 0.09168712049722672, 0.8121315836906433, 0.7076869606971741, -0.040020786225795746, 0.8328248858451843, 1.2409946918487549, -0.5012056827545166, 0.06808163225650787, -0.01105139497667551, -0.11220481246709824, 0.28963878750801086, -0.6597070097923279, -1.203682541847229, 0.14659984409809113, -1.1683369874954224, -0.7411877512931824, -0.1939469277858734, 0.5439397096633911, 0.21221284568309784, 0.5252321362495422, -0.6345597505569458, -1.078973650932312, 0.25034430623054504, 1.1844838857650757, 1.2168147563934326, 0.35125577449798584, 1.145604133605957, -0.05915190652012825, -0.7839270830154419, -0.5524666905403137, 0.33075928688049316, 1.1262937784194946, 1.4039596319198608, 1.5902118682861328, 0.7938192486763, -0.5998477339744568, -0.6860806345939636, -1.233276605606079, -0.616443395614624, -0.6037271618843079, -0.42942193150520325, -0.33977022767066956, 1.188004970550537, 0.9240955710411072, -0.5308518409729004, -2.4934608936309814, 0.49953293800354004, 0.08327615261077881, 0.7428743839263916, -0.19385899603366852, 0.5878033638000488, 1.6118416786193848, 0.11516346037387848, 0.4455083906650543, -1.0876981019973755, 0.5476579666137695, -2.1787655353546143, -0.6472018361091614, -0.07947126030921936, -0.9331670999526978, 1.0334434509277344, 0.14318948984146118, -0.3795817792415619, -0.6025420427322388, 1.0840784311294556, 0.19972071051597595, 1.424512267112732, -1.2781838178634644, 0.41088759899139404, 1.149430274963379, -0.24366508424282074, -0.07351594418287277, -0.28027546405792236, -0.41109931468963623, 0.04323556646704674, 0.5511549711227417, 1.5987578630447388, 1.6437846422195435, 0.787741482257843, 0.03752943500876427, -1.5119425058364868, -0.9445278644561768, 0.9118102192878723, -0.3743615448474884, 0.35502126812934875, 1.43047034740448, 0.6901648044586182, 1.1978821754455566, -0.6829984784126282, -0.5505804419517517, -0.8383521437644958], [0.8715062737464905, -1.3175400495529175, 1.6370761394500732, 0.5421773791313171, 0.18664735555648804, 1.9916387796401978, -1.022576928138733, 1.0825666189193726, 1.4582583904266357, 0.7968881726264954, -1.9109786748886108, -0.9277921319007874, -0.3030945360660553, 0.7391606569290161, -0.03862468898296356, -0.2547566890716553, -1.1918538808822632, -0.44054916501045227, 0.16635385155677795, 0.734446108341217, -1.726188063621521, -0.3169461190700531, 1.0646183490753174, -0.35742518305778503, -0.2273559421300888, -0.751714825630188, -0.3826776444911957, 0.18552468717098236, 0.13839642703533173, -0.3925158381462097, 1.0963271856307983, -0.9586095213890076, 0.20411869883537292, 0.11568310111761093, -0.324271559715271, -0.3949449956417084, 0.27977466583251953, -1.6956052780151367, 1.6833734512329102, 0.8124067187309265, -0.10195612162351608, 0.8863140940666199, -0.9720550775527954, -2.135305881500244, -0.45736050605773926, -0.18887539207935333, 0.9889172911643982, -0.10975630581378937, 0.7126286029815674, -0.5329652428627014, 1.425537109375, -0.6581724286079407, -1.4538323879241943, 0.7572633028030396, 1.158202886581421, 0.27236899733543396, 0.13145416975021362, 1.6392247676849365, 1.3516243696212769, 0.31634315848350525, 0.2301570624113083, 1.3568586111068726, -0.1232699602842331, -0.600740373134613, -0.06484336405992508, -0.07521750777959824, 1.3608319759368896, -0.825472354888916, 1.1126859188079834, 0.8112237453460693, 0.6792923212051392, 0.8991660475730896, 0.05075932294130325, -1.1129647493362427, -0.2826981842517853, -0.8732864260673523, 2.207052230834961, 0.8264896273612976, 0.004233148880302906, -0.782017171382904, -1.8772332668304443, 0.4524040222167969, 0.13598622381687164, -0.9197794198989868, -0.48429080843925476, 1.4300512075424194, 1.0955396890640259, 1.7627559900283813, -1.9364577531814575, 0.850395679473877, 0.281414270401001, 0.566749632358551, 1.2259427309036255, 0.9698328375816345, -0.6877927184104919, 1.9315193891525269, 0.09927931427955627, -0.4493911564350128, -0.017588220536708832, 0.7056578397750854, -0.07078393548727036, 1.2136907577514648, 1.7399022579193115, -1.050625205039978, 0.6847200989723206, 0.16308905184268951, 0.7616716027259827, -1.085155963897705, -0.6609418392181396, 1.047961711883545, -1.0586408376693726, -0.4579004943370819, 0.4034178555011749, 1.0093237161636353, -1.72940993309021, -0.7884783744812012, -0.4386357367038727, 0.11994282901287079, -0.006877140142023563, -0.20199517905712128, 0.4608834385871887, 0.424278199672699, 0.4049798846244812, -0.36043545603752136, 0.8132514357566833, 1.5893092155456543, -0.354696124792099, -1.3396048545837402], [1.820168375968933, -0.0707668736577034, -0.25467851758003235, 0.4024721086025238, -2.4004437923431396, 1.3540302515029907, -0.691010057926178, -0.010712161660194397, -0.1769842952489853, -0.2070886492729187, 0.718636691570282, 0.7977895736694336, 3.0227911472320557, 0.5511413216590881, -0.10710360109806061, 0.04484002664685249, 0.1906156837940216, -1.6320772171020508, 0.3917865455150604, 0.17987750470638275, 0.3374136984348297, -1.0136135816574097, -0.3104332387447357, 1.7297990322113037, -0.3206518888473511, 1.0080963373184204, 0.8290652632713318, 1.6849536895751953, -0.27257877588272095, 0.460332453250885, -0.27738118171691895, -1.6368098258972168, 0.9057375192642212, -0.9826779365539551, -0.49246349930763245, 1.3855931758880615, 1.7365810871124268, -0.7705590128898621, 0.24344958364963531, -2.233228921890259, 0.2601473927497864, -0.8957046866416931, 0.3490714728832245, 0.21534204483032227, 0.5174620151519775, -0.25036191940307617, 0.058466095477342606, 0.01818733662366867, 1.5756468772888184, 0.1195433959364891, -0.9917546510696411, -1.0851433277130127, 0.11223248392343521, 0.9349345564842224, -0.06281055510044098, -0.7593872547149658, 0.05916199833154678, 1.1255784034729004, 0.5911614298820496, -0.5321545600891113, -0.014423223212361336, -0.6762651801109314, 0.31475353240966797, -0.6777457594871521, 0.2084219604730606, -0.9367153644561768, 0.05901110917329788, -0.42751941084861755, 0.42705851793289185, 1.1960558891296387, 0.39941084384918213, -0.31121718883514404, 1.4585767984390259, 1.760290503501892, 0.6311680674552917, 0.42828813195228577, -0.5243129134178162, -1.1931637525558472, -1.7373178005218506, -0.6557032465934753, 0.9984699487686157, -1.3509434461593628, 0.21694409847259521, 0.95582115650177, -0.03438267484307289, 0.6287845969200134, 1.3698859214782715, -1.2755669355392456, 0.023594951257109642, -1.3388715982437134, -1.4411728382110596, 0.8888623714447021, -0.3349681496620178, 0.5393149852752686, 0.9141398668289185, -1.6431443691253662, -0.7106870412826538, 1.233816146850586, 1.5586107969284058, -2.3038549423217773, -1.5026532411575317, 0.6860861778259277, -0.03961636498570442, 0.4221007227897644, 0.9818860292434692, 1.916542410850525, -0.7596181631088257, -1.3053674697875977, 0.3930203318595886, 0.6988825798034668, 0.426146537065506, -1.192634105682373, 0.18855993449687958, 0.3321824371814728, -0.2201359122991562, -0.7675288915634155, 0.2933923602104187, -0.9125947952270508, 1.3204575777053833, 1.597546935081482, -1.2830462455749512, -1.5502351522445679, -1.854306936264038, -1.9087834358215332, -0.8109269738197327, 0.03343987837433815, 0.7232097387313843, 2.369797706604004], [1.1376540660858154, -0.6034818887710571, -1.2884302139282227, 0.15406568348407745, 0.6875466108322144, -0.9720901250839233, 1.5042890310287476, -0.7337709069252014, -0.022290684282779694, 0.711497962474823, -0.6867437362670898, -1.225076675415039, 0.3735334277153015, -1.1739424467086792, -0.7795138359069824, 0.6670461893081665, -1.5579020977020264, 1.491290807723999, -0.8441088199615479, 0.39991438388824463, -1.804611325263977, 0.6555398106575012, 0.23030725121498108, -1.3681061267852783, 0.9853777885437012, 1.3543765544891357, -0.4946952760219574, 0.9193693995475769, -0.5248844623565674, 2.1469526290893555, -0.9024722576141357, 0.31816011667251587, -0.32833513617515564, 0.5968291759490967, 0.7828689813613892, -0.03255925327539444, -0.6510273814201355, 0.9690494537353516, 0.4427156150341034, -1.1574835777282715, 3.1499197483062744, 0.6978086829185486, 3.781775951385498, -1.7557156085968018, -0.036147862672805786, -0.9553422927856445, 0.04241032525897026, -2.16715669631958, -0.3553953170776367, 0.7403756976127625, 1.2835661172866821, 0.11749816685914993, -0.25689375400543213, -0.8138211369514465, 0.48333483934402466, 0.4362501800060272, -1.5030807256698608, 0.08689998090267181, 1.6066824197769165, -1.1586463451385498, -1.704430341720581, 1.5411858558654785, 1.2684015035629272, -0.038409046828746796, -0.5393103361129761, 0.8134520649909973, -0.939060389995575, 1.0272973775863647, 1.2271982431411743, -0.44087904691696167, -0.7776392102241516, -1.1036241054534912, -0.25003471970558167, 0.8940643668174744, 0.3703485429286957, -0.7435706257820129, 0.34606459736824036, -0.7029657363891602, 0.37706732749938965, 1.5547163486480713, 0.9072831869125366, 0.46696364879608154, -0.6179125905036926, -2.114412784576416, -0.5963342785835266, 0.04731389880180359, -1.1602507829666138, -0.6698991060256958, -0.24252831935882568, -0.972852349281311, 1.4459521770477295, -0.1825844943523407, -0.5105578303337097, -0.6668713092803955, 0.15001507103443146, 1.2060807943344116, -0.7473549842834473, -0.06368411332368851, 1.1456724405288696, -0.041288018226623535, -2.637052059173584, 0.5647737979888916, -0.4663754403591156, 1.284095287322998, -0.38282445073127747, 1.427431344985962, -0.6901445984840393, 0.18212872743606567, -0.403426855802536, 0.27931490540504456, 0.2510894238948822, 0.7941832542419434, -0.637097179889679, -0.463369220495224, -1.4830422401428223, 0.45855483412742615, 0.24321617186069489, -1.3895145654678345, -1.0839431285858154, -0.7044283747673035, -1.295523762702942, -1.0538326501846313, 0.5206286311149597, 0.8636967539787292, 0.39185985922813416, -0.4531162977218628, -0.030139410868287086, -1.1707019805908203], [-0.18948574364185333, 0.30822113156318665, 0.054810427129268646, 1.2883121967315674, -0.6823484897613525, 1.2800978422164917, -0.04279748350381851, 0.8500811457633972, -0.37972956895828247, -0.11155404895544052, -1.2723745107650757, 0.8831928372383118, 1.2952098846435547, 0.20756137371063232, 0.1500220000743866, 0.395466148853302, 0.2235954850912094, -2.438898801803589, -0.19920167326927185, -0.04543021693825722, -0.1255202740430832, -0.9043362736701965, -1.3668372631072998, 1.2092291116714478, 0.4417630136013031, -0.17494069039821625, -0.4982164800167084, -0.07829615473747253, -0.34879621863365173, 0.6512877941131592, 0.15909580886363983, -1.8927934169769287, -0.9154860973358154, -0.841092050075531, 1.5390944480895996, -1.6585791110992432, 0.701602041721344, -0.3673284351825714, -0.22469528019428253, -1.0694912672042847, -0.7738832831382751, -0.6756041049957275, -0.7270308136940002, -1.7224489450454712, -1.396767497062683, -0.40184301137924194, -0.5827367305755615, -0.01659591682255268, -0.22008900344371796, -0.08560563623905182, -0.30907508730888367, 1.110887050628662, 0.8838918209075928, 0.5656027793884277, -0.026503104716539383, -0.4002346396446228, 0.46530020236968994, -0.38812097907066345, -0.0179912019520998, -0.7569178938865662, 0.8649154305458069, -0.23549588024616241, 0.3102570176124573, 0.9556566476821899, 1.413002848625183, 0.22126269340515137, 0.21891078352928162, 1.5466347932815552, 0.607022225856781, -0.09479586035013199, 1.1270431280136108, -1.57645583152771, 0.924535870552063, 0.7885586619377136, -1.1455694437026978, 0.8001826405525208, -0.7997161746025085, 0.72452312707901, -0.5222387909889221, 0.49850109219551086, -1.0896031856536865, -0.8291086554527283, -0.12561775743961334, 0.3388441503047943, -0.7398838996887207, 1.1532323360443115, 1.1773202419281006, -0.5418291091918945, -1.3317641019821167, 0.35892006754875183, -0.14606837928295135, -0.40827351808547974, -0.10513468831777573, 0.6726971864700317, 0.28134992718696594, -0.5064086318016052, 0.8982909321784973, 0.27907273173332214, 0.5600299835205078, -1.4674164056777954, 0.3395017385482788, 0.3187587261199951, -1.3654946088790894, 0.011819870211184025, -0.39121559262275696, -1.7980157136917114, 0.15512534976005554, -0.05496175214648247, -0.5339733362197876, 0.5603838562965393, -0.1952059268951416, -0.09641309827566147, -0.7755714058876038, 1.4676169157028198, -0.9658129215240479, -1.5384364128112793, 1.182565450668335, 1.328738808631897, -2.1173973083496094, 0.1190357506275177, 0.240342915058136, -2.0092287063598633, -2.050488233566284, -0.29277706146240234, -0.3494794964790344, 0.9267983436584473, -0.240866556763649, 0.12299732863903046], [0.17776912450790405, 2.3895161151885986, -1.055290699005127, -0.45513513684272766, -0.027766551822423935, -0.4633720815181732, 0.10441424697637558, 0.16247965395450592, -0.7995648384094238, -0.2078484147787094, -0.6655663251876831, 1.833764672279358, -0.3778854012489319, -1.4617139101028442, -0.8294524550437927, -0.7242216467857361, 0.7924299836158752, 1.5832908153533936, -1.2827410697937012, -0.4714401960372925, 1.1668115854263306, -0.2769070565700531, 0.3385544419288635, -1.0324416160583496, -0.00988311879336834, -0.49244093894958496, 0.20965972542762756, -0.1967393308877945, 0.09233657270669937, -0.8169274926185608, -0.9034377336502075, 1.15727961063385, 0.12959176301956177, -1.1969808340072632, -2.5584185123443604, 0.4165763854980469, -0.6175663471221924, 0.36593571305274963, -0.6371170282363892, -0.2216181457042694, 0.32125431299209595, 0.4393775463104248, -1.0328770875930786, 0.5259102582931519, 2.085136651992798, -1.1810498237609863, 0.7046926617622375, -0.1360093057155609, 1.4196120500564575, 0.2178807258605957, 1.3316410779953003, -0.751143217086792, -1.4459903240203857, -0.027977369725704193, -0.38633471727371216, -0.7328239679336548, -1.3834681510925293, -2.7392208576202393, -1.4943565130233765, 0.8198626041412354, -0.083010733127594, -0.33433252573013306, 1.0972753763198853, -0.6648820638656616, -0.4744780659675598, 0.5122607946395874, -1.0975538492202759, 0.12212882936000824, -0.30857494473457336, -2.358596086502075, -0.6984840035438538, 0.12755247950553894, 0.7734882831573486, -0.3094198703765869, 2.0620455741882324, -0.6182076334953308, 0.034948259592056274, 1.5668354034423828, -0.183770552277565, -0.4300706386566162, -0.42873984575271606, -0.3217345178127289, 0.42014241218566895, -0.9536571502685547, 0.5400810837745667, -2.1048409938812256, -0.4773305654525757, -1.0258620977401733, -0.30391600728034973, -1.0688530206680298, -0.2682466506958008, -1.1304731369018555, -1.3730953931808472, -0.12178938835859299, 0.7776108384132385, 0.147389754652977, 1.662983775138855, 1.0477370023727417, -0.11526480317115784, 0.6091978549957275, 0.9966342449188232, -1.8415991067886353, 0.6799174547195435, 2.07421875, 1.232797384262085, -0.5343847870826721, -1.461571455001831, -0.7536287307739258, -0.4847158193588257, 2.2738194465637207, -0.5732535123825073, 0.7412484288215637, 0.7478505373001099, -0.06623076647520065, -0.21924284100532532, -0.37297388911247253, -0.693912923336029, -1.9333076477050781, -1.081756353378296, -0.8563147187232971, 0.9080471396446228, -0.17435596883296967, 0.2888612747192383, -0.6860920190811157, -0.6014222502708435, -1.2467563152313232, -0.7877790927886963, 0.33905962109565735], [-0.05988926813006401, 0.5231543779373169, 1.4274128675460815, 0.16722019016742706, -0.43906131386756897, -2.923215866088867, -0.5557756423950195, 0.5479412078857422, 1.2716065645217896, -0.20943160355091095, 0.07129937410354614, 0.6119772791862488, 1.031514048576355, 0.20050367712974548, -1.7110339403152466, 1.351553201675415, 0.658453106880188, -1.3123958110809326, 1.442399263381958, -0.025706065818667412, -0.0487249530851841, -0.5208265781402588, 0.1045348271727562, 1.053878903388977, 0.657646119594574, 1.0485504865646362, -0.23755799233913422, -0.6860616207122803, 0.7592828273773193, -0.7883734703063965, -0.8810468912124634, -1.1592262983322144, 0.6445372104644775, 0.10540635138750076, -1.0646077394485474, 2.8620989322662354, 1.2532386779785156, -1.1436847448349, -0.3830706477165222, 0.1112021878361702, 1.4023633003234863, 0.5648852586746216, 0.15138423442840576, -1.4154685735702515, -0.5445226430892944, -1.1083664894104004, -0.5857659578323364, 0.21613991260528564, -0.1844809353351593, -0.7877517938613892, -0.003965287934988737, 0.739793062210083, -1.128381609916687, -1.9455641508102417, 0.8248536586761475, 0.16754750907421112, 0.1683356910943985, 0.8658910989761353, -1.936401605606079, -1.6120338439941406, 0.17342442274093628, 1.668975591659546, 0.03252840414643288, 0.26776495575904846, -0.8360867500305176, -1.519095778465271, 1.0463993549346924, 0.12113217264413834, -0.3510369062423706, -0.4318126440048218, 1.6805334091186523, 0.386962890625, 0.8485475182533264, 0.17077282071113586, -1.303882360458374, -0.9285829067230225, 0.34569573402404785, 0.2866855263710022, 1.8899751901626587, 0.0651472806930542, -0.25908032059669495, 1.7636055946350098, -0.9554212689399719, 0.9208016991615295, -0.377811998128891, 0.1279267966747284, -0.7198097109794617, 1.0711101293563843, 0.9856600761413574, -0.3697631359100342, -1.3279997110366821, 0.5570911169052124, 1.0978972911834717, 0.4412957429885864, 0.6087859272956848, -0.20489269495010376, 0.25797978043556213, -1.6649342775344849, -0.9204759001731873, 1.5123238563537598, -0.30334243178367615, -1.6499438285827637, -0.7081895470619202, -1.500037431716919, 1.0717447996139526, 0.5026458501815796, -0.9593538641929626, 0.2596290707588196, -0.23629999160766602, 1.2233744859695435, 0.04904267564415932, 0.8077335357666016, 1.5104496479034424, 0.07260072976350784, 0.7192867994308472, 1.163426399230957, -1.8998737335205078, 0.6831393837928772, -0.14294122159481049, 0.6145243048667908, 0.010543158277869225, 0.14866751432418823, 3.478959560394287, -2.90230655670166, 0.17248061299324036, -2.251016139984131, 0.5156617760658264, 0.944976270198822], [1.4716846942901611, 1.4845832586288452, -0.08669506013393402, -2.0283420085906982, 0.680015504360199, 0.7638290524482727, -0.02050371654331684, -0.0005722492933273315, -0.7281249761581421, -0.028459642082452774, 1.6684032678604126, -0.1727268099784851, 0.3423306345939636, 0.9667173624038696, -0.6969346404075623, 1.251862645149231, 0.3240678310394287, 0.1791934370994568, -0.7819766402244568, -1.4988220930099487, -1.348629355430603, 1.4141442775726318, 0.2507181763648987, -0.4664832055568695, -1.4711048603057861, -0.22459860146045685, -0.4116627871990204, -0.964209258556366, -1.2169140577316284, -1.9721394777297974, -2.1557726860046387, 1.791068434715271, -0.8396593928337097, -0.33757925033569336, 1.543034315109253, 0.1495334804058075, -0.7455114722251892, 0.7819620966911316, -1.0203511714935303, 2.025641918182373, -0.8404316902160645, 0.5012148022651672, 0.4565434157848358, 0.0923333391547203, -0.6268941760063171, 0.8461753129959106, -1.170876145362854, 1.5421048402786255, 0.12784820795059204, 0.9134012460708618, 2.020139217376709, 0.18614818155765533, -1.66213858127594, 0.774086594581604, -0.2493194043636322, -0.8295220732688904, 2.146768093109131, 0.6443893909454346, -1.354212999343872, 0.20361803472042084, 1.0765937566757202, 0.9187750816345215, 1.173740029335022, -0.10857474058866501, 0.9165105819702148, -0.629435658454895, -0.06574023514986038, -0.8910971879959106, 0.7597147822380066, -0.4807300269603729, -1.601645827293396, -1.5504722595214844, -0.46087872982025146, 0.25962358713150024, -0.17944251000881195, -0.14949916303157806, -1.57310152053833, 3.1292128562927246, -1.4526851177215576, 0.6901199221611023, -0.3653869926929474, 0.1227836012840271, -0.7967063784599304, -0.9234988689422607, -0.25838544964790344, -2.485327959060669, -1.0135146379470825, -1.3303136825561523, 1.7218513488769531, 0.5807791948318481, -1.035866379737854, -0.5550994277000427, -0.9909635782241821, 0.11117614805698395, -1.1220592260360718, 1.2247791290283203, -1.8070836067199707, 0.32895922660827637, -0.16980800032615662, -1.351251482963562, 1.5923480987548828, 1.0731168985366821, 0.4312928020954132, 1.5536326169967651, -0.8639686107635498, -0.3463495075702667, 0.33549126982688904, 1.4883084297180176, 0.4619317948818207, -0.25821739435195923, 0.601860761642456, 0.9554150104522705, -0.020624283701181412, -0.48422491550445557, -0.7338101267814636, 0.8811483383178711, -0.29899832606315613, -0.2230253964662552, 0.2427448332309723, 0.2717570662498474, -0.34658050537109375, -0.41459542512893677, 1.2156459093093872, 1.2506415843963623, 0.1712133139371872, 1.1285864114761353, -1.3713058233261108, 0.7876591086387634], [0.11106737703084946, 1.7869770526885986, -0.13116881251335144, -0.7104580998420715, 0.12516675889492035, -0.6277387142181396, 0.5307325720787048, 1.2749121189117432, 1.047121286392212, 0.7565120458602905, -2.270214796066284, -0.9512282013893127, -1.1484090089797974, -0.18362465500831604, -1.3258508443832397, -0.68259596824646, -0.5790307521820068, 2.4157986640930176, 1.3184576034545898, 1.6747033596038818, -0.402829110622406, 0.2697761058807373, -0.8367900252342224, 0.5042166113853455, 0.6843888759613037, 0.7965939044952393, -0.5131798982620239, 1.422183632850647, -0.9624855518341064, 0.6663969159126282, 0.9708598852157593, -0.07261078804731369, -0.8937373757362366, -0.3792210817337036, -0.10577202588319778, 1.3435508012771606, -0.3872847259044647, 0.18334956467151642, 0.8763569593429565, 0.5089400410652161, -0.8065299987792969, -0.2394048422574997, -0.6130708456039429, -0.6664157509803772, 0.02883049100637436, -0.4042493402957916, -0.1512996107339859, -2.2156496047973633, -0.3637043237686157, -2.0666379928588867, 0.5735595226287842, 0.34856918454170227, 1.183755874633789, 0.07708024233579636, -2.097820281982422, 0.6403185725212097, -1.6577335596084595, 0.31686824560165405, 1.313991665840149, -0.8653044700622559, 0.19384922087192535, 1.0643010139465332, -0.19191096723079681, -0.7313897013664246, -0.33180779218673706, 0.8634229898452759, 0.02131778746843338, -0.3082412779331207, -0.4071989953517914, -0.2645839750766754, 0.7008068561553955, 0.15566805005073547, 0.0013838243903592229, -0.07231758534908295, -0.9724782705307007, 1.5412218570709229, 0.8185646533966064, 0.4149079918861389, -0.03351251780986786, 0.11559057980775833, 0.4187508523464203, 1.1871933937072754, -0.5206015110015869, 0.49946853518486023, 0.4892825782299042, -0.9021301865577698, -0.012896297499537468, -0.269275039434433, 0.34327420592308044, 0.7777602076530457, 2.7278473377227783, -1.242989182472229, -0.6698564887046814, 0.2361166924238205, -0.30447548627853394, 0.8948262333869934, -1.624469518661499, 0.785784125328064, -0.3958907723426819, -0.032741695642471313, 1.4716582298278809, 1.0020833015441895, 0.3933779001235962, -0.8063941597938538, 0.04226721078157425, 0.4884130358695984, 0.014485841616988182, 1.0120289325714111, -1.831641674041748, 0.015859879553318024, -1.6931313276290894, -0.5219823122024536, -1.0513004064559937, 1.1543166637420654, 1.4224170446395874, 1.734694242477417, -1.4548523426055908, 1.278626561164856, -0.6631287336349487, -0.41368016600608826, -0.23287975788116455, -0.6078081727027893, -0.1180124282836914, -0.3812750279903412, 0.7836549878120422, -0.5444105267524719, 0.8660091757774353, -1.4995944499969482], [-0.060518212616443634, -1.056206464767456, 0.15342000126838684, 0.15285086631774902, -1.0166289806365967, -0.7779886722564697, -0.9843139052391052, 0.546294093132019, 0.26957637071609497, 0.5770278573036194, -1.4160619974136353, 0.6221336722373962, 0.863828182220459, -0.030095916241407394, -0.5294134616851807, 2.1984963417053223, 0.09827693551778793, -0.24720558524131775, -0.5760577321052551, -0.3765479326248169, 3.2180099487304688, 0.7442073225975037, 0.4425537884235382, 1.9569413661956787, -0.16692252457141876, 0.05899101868271828, 1.094387173652649, -0.7031165361404419, 1.5610703229904175, 0.686904788017273, -0.15871337056159973, -0.12407969683408737, 0.05037861317396164, -1.2792222499847412, 0.9926387071609497, -0.9553432464599609, -0.23893208801746368, 1.0630820989608765, 1.266214370727539, 0.16217392683029175, -1.4134174585342407, 0.9477100968360901, -0.5054629445075989, -0.6415577530860901, -0.782680869102478, -0.2589575946331024, -0.814942479133606, 0.46606308221817017, 1.118006706237793, 1.3047728538513184, 2.038480281829834, 0.3202807605266571, -0.5221448540687561, 0.33147716522216797, -0.3900661766529083, -0.3017476499080658, -0.7219057679176331, 1.3196308612823486, -1.2076495885849, 0.3859298825263977, 0.4886021912097931, 0.556134819984436, -0.4190714657306671, -0.8637099266052246, -1.0067064762115479, 0.42256948351860046, 1.276949167251587, 0.22688063979148865, 0.0323239229619503, 1.443990707397461, -0.4001557230949402, 0.3340529203414917, -1.2304465770721436, -1.0690453052520752, 1.2963409423828125, 0.08036237955093384, -0.1333652287721634, -0.8105769753456116, -1.2500473260879517, -0.9793378114700317, -2.2564752101898193, 0.175296351313591, -0.8146049380302429, 0.10764462500810623, -1.3433637619018555, -1.8049466609954834, -0.6903793811798096, -0.4007912278175354, 0.323201060295105, -1.1226346492767334, 1.131441354751587, -1.3611063957214355, -1.9082841873168945, 0.5194631814956665, -0.14651791751384735, 0.3775346279144287, 0.9096692204475403, 0.20091629028320312, -1.367302417755127, -1.3625637292861938, 1.0632270574569702, -0.7170004844665527, -1.0168401002883911, -0.7649524807929993, -1.367279291152954, 1.5578404664993286, -0.8738435506820679, -0.7958239912986755, -1.8366297483444214, 1.0345767736434937, -1.4565590620040894, 0.11564969271421432, 0.14017032086849213, -0.27753543853759766, 0.20629532635211945, 0.0930512398481369, -0.7138401865959167, -0.7693091630935669, -0.5041179656982422, -0.09078515321016312, -1.103629231452942, -0.41070979833602905, -0.021205805242061615, 0.18475839495658875, -1.8350828886032104, -0.34652620553970337, -1.3582525253295898, -2.4251785278320312], [-0.3101300299167633, 0.6631854176521301, 0.2087571918964386, 1.601517915725708, 0.817521870136261, 0.22343523800373077, 1.2122117280960083, -0.8358983397483826, -0.9627358913421631, 0.9361781477928162, 0.31737375259399414, -0.06599650532007217, 0.21350708603858948, 0.3971911072731018, -0.6835360527038574, 0.563810408115387, -1.5637853145599365, -1.2630521059036255, -0.008717726916074753, 0.13932235538959503, 0.291116327047348, -0.9886721968650818, 0.31004759669303894, -1.8231061697006226, 0.20107433199882507, -0.6376466155052185, 0.9118244647979736, -1.7991759777069092, -0.9216150641441345, 0.3271224796772003, -0.6903219223022461, 0.9249876141548157, -1.174019694328308, 0.16307486593723297, 0.04304670915007591, 0.8452777862548828, -0.42974334955215454, 0.9990893006324768, -1.2691150903701782, 0.14661721885204315, 0.4503204822540283, -0.2142946869134903, -0.6422017812728882, -0.5587930679321289, 0.1625024974346161, -1.0610986948013306, 0.311908483505249, 0.9720500111579895, 0.8232330083847046, -0.5014359951019287, -0.525951623916626, 0.9001634120941162, -0.6205838322639465, 1.635674238204956, 1.6532444953918457, -0.24078023433685303, -0.12596823275089264, 0.42118746042251587, 0.6845080256462097, -2.024071455001831, 1.0316139459609985, -1.4077427387237549, -0.7138276100158691, -0.4124480187892914, 0.40769466757774353, 0.041211988776922226, -1.4103869199752808, 0.7870243191719055, -0.7318472266197205, 1.7838557958602905, 1.9177861213684082, 0.7394888401031494, 1.0888807773590088, 0.6055111885070801, 0.2323356568813324, -1.1043412685394287, -0.27737629413604736, 0.9442505240440369, -0.20020072162151337, -2.702563524246216, -1.6550984382629395, -0.9854453802108765, -1.0627912282943726, 0.040035489946603775, 0.23187704384326935, -1.6514703035354614, -0.210840106010437, -1.15264093875885, -0.16288688778877258, 0.8145295977592468, 1.2334752082824707, -0.2977639436721802, -0.8536803722381592, 1.0929309129714966, -0.8765704035758972, 0.06545404344797134, -0.751453161239624, 1.2947226762771606, 1.1259640455245972, -0.9297978281974792, 1.9271912574768066, -0.8746350407600403, 0.6877759099006653, -1.1398406028747559, 0.18225979804992676, 0.11636906117200851, -0.7359537482261658, -1.3325785398483276, 0.027999501675367355, 1.2383111715316772, 0.27975592017173767, 0.2249767780303955, -0.6186845898628235, 1.4839868545532227, 0.10818099230527878, 1.4252598285675049, -0.6111276149749756, 1.0867325067520142, 0.08137331902980804, 1.3356270790100098, 2.2262487411499023, -0.9316822290420532, 0.18618011474609375, 0.5212445855140686, 1.3895432949066162, -0.28155919909477234, 0.7915396094322205, -2.083704948425293], [1.0301799774169922, -0.7400140762329102, -0.822172224521637, -0.0840267464518547, 0.20189547538757324, 0.2555558979511261, -0.10484001040458679, -1.515927791595459, 0.059885043650865555, 0.2032601535320282, -0.39569658041000366, -2.3894307613372803, -0.50563645362854, 0.3826411962509155, 0.09280771762132645, 0.9628977179527283, -1.2043629884719849, -0.8497493863105774, -0.46214497089385986, -0.8694173097610474, 0.07825113832950592, 0.9760456681251526, -0.8920902609825134, 0.8399397134780884, -0.4829384982585907, -0.7033441066741943, 0.04531610757112503, 2.4229910373687744, -0.3034931421279907, 1.7088685035705566, 0.17362312972545624, -0.18996869027614594, 1.895009994506836, -0.47207003831863403, -1.2048366069793701, 0.1927906572818756, 0.8793150186538696, -1.230178713798523, -0.5723007321357727, -0.7778531312942505, -0.00926243420690298, -0.6212483644485474, -0.02151459828019142, -0.8306427001953125, 1.753164291381836, -0.4704115390777588, 1.6316922903060913, -0.24710369110107422, -2.938572645187378, -1.8289296627044678, 1.7049384117126465, -0.5873175859451294, 0.23484350740909576, 0.43146398663520813, -0.2556774616241455, -1.387963056564331, -0.5861356258392334, -0.8868626952171326, -2.238851308822632, 2.2564079761505127, 0.6158797740936279, 1.0926172733306885, -1.073486089706421, -1.8027454614639282, 0.3534221649169922, -0.519457221031189, 0.9085900187492371, 0.06013231724500656, 0.9282834529876709, -0.6009721755981445, -0.00709601491689682, -1.0449557304382324, 0.4274618923664093, -0.3213721811771393, 0.49147412180900574, -0.7646270990371704, -1.0194263458251953, 0.009385556913912296, -0.45941463112831116, 1.3572783470153809, 0.69646817445755, -2.030139207839966, 1.7065937519073486, -2.079782724380493, 0.4769788682460785, -0.3978525698184967, -0.7165222764015198, 0.17099344730377197, 0.002722229575738311, 0.08392815291881561, -1.6755210161209106, 0.6336332559585571, 1.8081804513931274, 0.8854008316993713, -0.34573787450790405, -0.8353874683380127, -0.3375920355319977, 2.4277889728546143, -0.7206687927246094, 1.2082489728927612, -1.606102705001831, 0.1658933460712433, 0.04289838671684265, 0.809018611907959, -0.2518821954727173, -1.1936689615249634, 0.8821926712989807, 0.38040274381637573, 1.8620061874389648, -0.524257481098175, -0.5495148301124573, 0.1399405300617218, 1.9482592344284058, -0.40323755145072937, -0.21333952248096466, -0.11516263335943222, 0.2361898124217987, -0.21407251060009003, -0.050047386437654495, 1.3370659351348877, 0.24420225620269775, 0.6314799785614014, -0.980474591255188, -0.2926759123802185, -0.48346075415611267, 0.25928637385368347, -0.8167484402656555, -0.05915411189198494], [-0.3879848122596741, -0.5879697799682617, 0.5293604731559753, -0.9345230460166931, 0.4432421028614044, -0.5464286804199219, -0.5754785537719727, -1.254365086555481, 1.6012365818023682, 0.1132996529340744, 0.49537086486816406, 1.3504177331924438, -2.0028021335601807, 0.3602105379104614, 0.38825279474258423, -0.6796888113021851, -1.104141354560852, 0.07439802587032318, 2.4259636402130127, 0.2513110637664795, 0.6006295680999756, -0.38698849081993103, 0.0009053311659954488, -0.22113773226737976, -0.054929427802562714, -0.9208909869194031, 0.4800289571285248, -0.7353895902633667, 0.7927825450897217, -1.123674750328064, -0.8062000870704651, 0.08743791282176971, -0.6956768035888672, 0.5083842277526855, -0.04998611658811569, -0.0361294187605381, 0.193067267537117, 0.8421128392219543, 0.4402337372303009, -0.035256605595350266, -0.008772596716880798, 1.9283499717712402, 1.19166898727417, 0.34332388639450073, -0.67363440990448, -0.3681989312171936, -0.853839099407196, 0.984233558177948, -1.1979162693023682, -0.33698466420173645, 0.7689454555511475, 0.6903612613677979, -0.7944455146789551, 2.177036762237549, 1.7300949096679688, -1.7545264959335327, 1.5701042413711548, 1.9746750593185425, -0.9217491149902344, -0.08031627535820007, -0.17537134885787964, 0.9776164889335632, 1.6967357397079468, -0.138260617852211, 2.147970676422119, 0.7154530882835388, 1.1867023706436157, 0.7110153436660767, 0.18438170850276947, 0.6474334001541138, -0.15430429577827454, 1.8555940389633179, 2.470207929611206, -0.4992049038410187, -1.3058946132659912, 0.4044516682624817, -0.19813090562820435, 0.7591369152069092, -0.29130470752716064, -1.0300979614257812, 1.0891722440719604, 0.9922879934310913, -0.31305500864982605, -0.9130800366401672, -0.2709265947341919, -0.1906490921974182, -0.09654969722032547, -1.7337636947631836, 0.341012567281723, -1.1629573106765747, 1.1618703603744507, -0.3246511220932007, 1.1575464010238647, 0.7501652836799622, 1.2956595420837402, 0.24272951483726501, -0.3540373146533966, 0.04093320295214653, -0.3504442274570465, -1.5908681154251099, 0.9860070943832397, 0.3655480146408081, -0.1784578114748001, 0.34466153383255005, -0.37749606370925903, -0.9016849994659424, 0.545811116695404, -0.8033231496810913, -1.27894127368927, -1.7308661937713623, 0.9062690138816833, 0.28842857480049133, -0.8714375495910645, -0.7734991312026978, 0.0812717005610466, 3.019613027572632, -0.5174944400787354, 2.1921799182891846, -0.8862898945808411, 0.06847234815359116, 0.23528575897216797, 1.4897234439849854, 0.4495720863342285, -0.41925349831581116, 0.8680533766746521, 0.44846221804618835, 0.12762197852134705, 1.2728054523468018]], [[-1.4283474683761597, 1.0831220149993896, 0.007567298598587513, -0.6704248785972595, -0.45061302185058594, -1.8638887405395508, -1.1088124513626099, 0.818952739238739, -0.22841356694698334, -1.4000834226608276, 0.650326132774353, 0.09046774357557297, -0.5417652726173401, 0.9309425354003906, 1.1862415075302124, 0.8365603685379028, 0.19467668235301971, 0.29299601912498474, -0.6684669852256775, -0.3866361677646637, 3.9376392364501953, -0.28109586238861084, -1.0577236413955688, -0.42143142223358154, -1.4530251026153564, 0.5717251300811768, -0.3023111820220947, -1.459928274154663, -0.412438303232193, 0.09686282277107239, -1.496863603591919, -0.6280257701873779, -1.5875858068466187, 2.006460666656494, 0.5909343957901001, -2.7872068881988525, -0.6999333500862122, 0.28397640585899353, -0.06219670549035072, -0.11493373662233353, 0.46214520931243896, -0.7020782828330994, -1.0315735340118408, -0.09063290059566498, 1.5719672441482544, 0.7017049193382263, 1.8810455799102783, 0.9839652180671692, 0.06643368303775787, 0.391260027885437, -2.060737371444702, -0.6897854804992676, -0.3281344175338745, -1.3524185419082642, 0.13714629411697388, -0.7111753225326538, -1.281527042388916, 1.2403011322021484, 0.3804946541786194, 0.47613099217414856, -1.0136741399765015, 2.159869432449341, 0.6848452091217041, 1.0287717580795288, -1.6386778354644775, 0.4772525727748871, 0.35792577266693115, -0.663530170917511, 0.30464306473731995, 0.607861340045929, -0.1923728734254837, -1.2008583545684814, 0.35934922099113464, -0.06460662186145782, 1.3896541595458984, 0.24320301413536072, -1.508735179901123, -2.2191708087921143, -0.6658815741539001, 0.3858279585838318, -1.011500358581543, 0.32462912797927856, -1.2211662530899048, -0.0762869343161583, 0.7503848075866699, -0.97520512342453, -0.2412053495645523, 0.6553575992584229, 0.8873063921928406, 1.0452970266342163, -1.0294475555419922, 1.7293404340744019, 1.5861918926239014, 0.5511841773986816, -0.30352911353111267, 1.1105698347091675, -1.0512886047363281, -0.9961901307106018, 0.3620765507221222, 0.07745544612407684, 0.5593163371086121, -0.08928569406270981, -1.0065752267837524, 1.0504170656204224, 1.4305403232574463, -1.8400170803070068, -0.5686086416244507, -0.16191831231117249, 0.7154256701469421, 0.670977771282196, -1.0482186079025269, 0.40476682782173157, 0.5980741381645203, -0.4995976686477661, -0.6586235165596008, -1.8638687133789062, -0.1656220555305481, -0.7204922437667847, -1.4414968490600586, 2.2991347312927246, 1.5020029544830322, 0.849474310874939, -0.6353751420974731, 0.006063912995159626, 0.08305349200963974, -0.7370128035545349, 0.037420812994241714, 1.4028135538101196], [-0.2748550474643707, 0.7261525988578796, 0.6987060904502869, 0.1651613414287567, 0.5718550682067871, 0.002339873695746064, -0.4984174370765686, 1.383484125137329, -1.4991246461868286, -2.1470911502838135, 1.843587875366211, -1.097101092338562, 2.1975655555725098, -0.17864862084388733, -1.791774034500122, -0.5138540267944336, 1.291983723640442, -1.698622465133667, -0.2652478516101837, -0.5112621188163757, 1.1091468334197998, -1.343095064163208, -1.5070693492889404, -1.0835626125335693, 0.500274658203125, -0.54137122631073, 1.1317733526229858, -0.2263329029083252, -1.204906702041626, 1.4928311109542847, 0.041005268692970276, -1.6893266439437866, -1.3039087057113647, -0.5225024819374084, -1.3079215288162231, -0.2956464886665344, -0.01913599483668804, 1.1847196817398071, -0.01791442558169365, 1.2105624675750732, 0.6405872106552124, -0.9179522395133972, 0.4995530843734741, -1.422498106956482, -1.0126370191574097, 2.277770757675171, 0.8171060681343079, -0.9263157844543457, -0.4919268786907196, -0.9492835402488708, -0.20918311178684235, 0.5302534103393555, 0.06363637000322342, -1.0263539552688599, -0.8330494165420532, 0.3610963523387909, 0.8495386838912964, -0.04751469939947128, -0.15752427279949188, -0.21253089606761932, -0.45046666264533997, -1.259134292602539, 0.14502264559268951, 0.20067238807678223, -0.2732752561569214, -0.9276608228683472, -1.3987467288970947, -2.5388433933258057, 1.0720772743225098, -2.4732589721679688, 0.10755913704633713, -0.6527605652809143, 1.1033812761306763, 0.7901930212974548, -1.87393319606781, 0.9163768887519836, 0.8657524585723877, 0.08117658644914627, -0.7064812779426575, -1.1340999603271484, 0.7545782923698425, -2.210150957107544, -1.1020840406417847, -0.9469968676567078, 0.6804123520851135, 0.4153953790664673, 0.02179880626499653, -0.6630627512931824, 0.45819351077079773, 1.3817179203033447, -3.253398895263672, -0.8993338346481323, 0.2747448682785034, 0.07010724395513535, 1.4170211553573608, 0.20635053515434265, 1.675998330116272, -2.601266860961914, -1.1098434925079346, -0.21365396678447723, 0.8584872484207153, -0.05666960030794144, -2.9765517711639404, 0.15284232795238495, -0.37821269035339355, 1.9929676055908203, 2.3579721450805664, -0.24770452082157135, -1.8581374883651733, -0.18170903623104095, -0.2964499294757843, -0.6118528246879578, -0.20404136180877686, -0.10888964682817459, -1.24214506149292, 0.07539896667003632, -0.28265833854675293, 0.0005165757029317319, -0.5463321805000305, -0.1841018795967102, -0.2858274281024933, -1.019502878189087, 0.5784997344017029, -0.420791357755661, -0.629482626914978, -1.2278623580932617, 0.9547656774520874, -0.022596167400479317], [-0.2812807261943817, -0.460716187953949, -0.7133038640022278, -0.4034349024295807, 0.5760881900787354, 0.07773400843143463, 1.354403018951416, -1.8523041009902954, 0.9667103886604309, -2.0070688724517822, 1.8358296155929565, 1.1150202751159668, -0.4929594397544861, -0.3094440698623657, -1.243959903717041, 0.5416337251663208, 0.3878230154514313, 1.1774705648422241, 1.135315179824829, -1.4730325937271118, -0.3339216113090515, -0.9597627520561218, -0.4121156632900238, -1.6783372163772583, 0.17717444896697998, -1.7520636320114136, -0.1800471693277359, 0.5873620510101318, -1.4612951278686523, 0.7142342329025269, -0.25542154908180237, -1.165810465812683, 1.7771227359771729, 0.24377764761447906, -0.36191898584365845, -0.6858904361724854, -0.6156489253044128, -0.9652677774429321, -0.48163220286369324, -0.21936677396297455, 0.7157776355743408, -1.0904862880706787, -0.4336549937725067, 0.7438775897026062, 0.15626740455627441, -0.06660757958889008, 0.8278766870498657, 0.8033081293106079, -0.1603546142578125, 0.5403344631195068, -0.9970043897628784, 1.8292968273162842, 0.5685335397720337, 0.6127650141716003, -0.2367691844701767, -1.277008056640625, 0.27846184372901917, 0.47368791699409485, -0.6273674368858337, -0.45486196875572205, -0.12658116221427917, 0.13110415637493134, -0.9653633832931519, -0.45652633905410767, -0.3178056478500366, 0.3526152968406677, 0.18652144074440002, -0.8942708969116211, -1.1378278732299805, 0.4231366217136383, -0.1476595401763916, -0.5266548991203308, -0.49681392312049866, 1.1029961109161377, 0.8116625547409058, 1.5593012571334839, -1.5171151161193848, 2.525585412979126, -0.8228410482406616, -0.6346836090087891, 1.505733847618103, -1.1869359016418457, 2.1190335750579834, 1.4074976444244385, 0.01255129650235176, -0.7418448328971863, 0.3186756372451782, -2.108311176300049, -1.4757951498031616, 1.2581411600112915, 0.10338499397039413, -0.8746140003204346, 2.187930107116699, -0.9228023886680603, -0.6235016584396362, 0.8660044074058533, -1.7108112573623657, 0.9055431485176086, 0.7160384654998779, 0.7744117975234985, -0.012736734934151173, 0.0234576053917408, -0.19739244878292084, 0.2291073501110077, 0.3603849411010742, -0.6246797442436218, 1.6475684642791748, -0.08409297466278076, 0.8485368490219116, 0.5754507184028625, 0.7452030181884766, -0.3487889766693115, -0.26942628622055054, -0.3712672293186188, 2.279146671295166, -0.7178519368171692, 1.5493123531341553, -0.9819775819778442, -1.0086171627044678, -0.13249537348747253, -0.3162126839160919, 0.6867967247962952, 0.1819727122783661, -1.950160264968872, 0.05497628450393677, 1.4928432703018188, -0.08352267742156982, 0.15567627549171448], [1.2804726362228394, 1.5670874118804932, -0.28361839056015015, -0.38798248767852783, -0.4379860758781433, -0.9461168646812439, 0.2892191410064697, -1.2721397876739502, 0.6962611079216003, 0.4869334399700165, 0.02679450996220112, 0.8388121128082275, 0.5284463763237, 1.7416284084320068, 1.290755033493042, 0.7848308086395264, -1.691786766052246, -0.5163753628730774, -1.7754783630371094, -1.3049581050872803, -1.004552960395813, 1.793541431427002, -0.2742253243923187, 0.37896546721458435, -0.48906734585762024, -0.4508046805858612, -0.6078041195869446, -0.32603195309638977, -0.4147247076034546, -0.616309404373169, -0.40159881114959717, -0.22994178533554077, 1.6579002141952515, -0.3113689720630646, -0.3350038528442383, 1.014892816543579, 0.7899511456489563, 0.18628518283367157, 0.5292956829071045, -0.6352908611297607, 2.725268602371216, -0.11939406394958496, 1.052656650543213, 1.6514556407928467, 0.9432389736175537, 0.8661684393882751, -1.0216785669326782, 0.8283771276473999, 0.6901783347129822, -0.5698168277740479, 0.4035249352455139, 0.6754706501960754, 0.252318799495697, -0.6226088404655457, 0.5269504189491272, -0.2956799566745758, -1.6282684803009033, 0.2484983503818512, 0.6236444711685181, 2.1531312465667725, 0.5396431684494019, 0.5622208118438721, 0.5922186374664307, -1.3217058181762695, 0.04723619669675827, 0.33598652482032776, -1.6493465900421143, -0.36562711000442505, 3.2815566062927246, 0.6375070214271545, -0.2952171862125397, 1.5211752653121948, 9.197987674269825e-05, -1.0117682218551636, -2.0300278663635254, 0.022219279780983925, -0.34676843881607056, -0.8945482969284058, 0.29196956753730774, -0.8938431143760681, 0.6785231828689575, 0.10387736558914185, -0.23485897481441498, -0.746505081653595, -0.532161295413971, -0.24283234775066376, -0.3641360104084015, 1.526107668876648, -1.1419541835784912, -1.0329244136810303, 1.3809764385223389, -2.6008036136627197, -1.0199522972106934, -0.36543071269989014, -0.32656314969062805, -2.002786636352539, -0.8769312500953674, 1.2592710256576538, 0.07298596948385239, -0.9688050150871277, -0.5512431263923645, 0.3852553069591522, 0.3222583830356598, 1.2200291156768799, 0.5640065670013428, 0.46332263946533203, 0.9686527848243713, -0.4240803122520447, 1.249031901359558, -1.0366175174713135, -1.0150011777877808, -0.3829331696033478, 0.36055731773376465, -0.6055535078048706, -1.0458204746246338, -0.1896822452545166, 0.3875049650669098, -0.1837398260831833, 0.8993557691574097, -0.5971378087997437, -0.3609413206577301, 0.41663357615470886, 0.4216513931751251, 0.483368456363678, 0.2703039050102234, -0.7918746471405029, 0.9226275682449341, -0.030927741900086403], [0.9919455647468567, 1.842614769935608, 0.09860209375619888, 0.6357336640357971, -1.0803622007369995, 1.2676178216934204, -0.9348496794700623, -1.644757628440857, -0.5210252404212952, 1.6182376146316528, 0.7428778409957886, 0.682266891002655, -1.031577467918396, -0.9715456366539001, -2.215473175048828, -1.8215211629867554, 0.6243005990982056, -1.6940703392028809, -0.3600347936153412, 0.7681946158409119, -0.5490999221801758, -0.8987985849380493, 0.04741397500038147, -0.2528076171875, -0.9264397025108337, 0.8237566351890564, 0.6278351545333862, 0.344419002532959, -1.4826200008392334, -0.6078395247459412, -0.10295356065034866, 1.345257043838501, 0.19206705689430237, -1.8738360404968262, 0.3716910183429718, 0.4453789293766022, -0.27558282017707825, 1.3203307390213013, -0.04103008285164833, 0.9231510162353516, -0.08063352108001709, -2.501079559326172, -0.692467212677002, 0.06208484247326851, 1.310003399848938, -0.4555794894695282, 1.1985948085784912, 0.7128579020500183, 0.0948539674282074, 0.08260639011859894, 1.0866882801055908, 0.1279640644788742, 0.24116148054599762, 1.0373879671096802, -0.3506854176521301, -0.39910149574279785, -1.359879970550537, 0.9208108186721802, -1.2155299186706543, -0.8481910228729248, 0.004676932003349066, -1.0852018594741821, 1.4981848001480103, -1.1017804145812988, -1.0454463958740234, -0.6363236904144287, -1.6434766054153442, 0.04500999301671982, 0.19185441732406616, -1.0987560749053955, -1.2827454805374146, -0.7239159941673279, -1.1839237213134766, -0.1671990305185318, 2.695291042327881, -0.01389367040246725, 0.16307885944843292, -0.08316415548324585, 2.3842129707336426, 1.241995930671692, -0.8142882585525513, -0.4346499443054199, 0.9172391891479492, 1.4408149719238281, -1.6460891962051392, -1.6616847515106201, -0.853570818901062, 0.8953693509101868, -0.1367194801568985, -1.7478524446487427, -0.5012921094894409, -0.16107137501239777, 0.8282551765441895, -0.10869855433702469, -0.8233973383903503, 0.5224446058273315, -0.16901031136512756, -1.3396576642990112, -2.7521679401397705, -1.1729962825775146, -0.6281500458717346, -1.9275813102722168, 0.7312303185462952, -0.526043713092804, -0.059357114136219025, 0.9082556962966919, -1.1960430145263672, -0.0834767073392868, 0.6483311653137207, 0.5177589058876038, 1.2936466932296753, -1.519906997680664, 0.9412474036216736, -0.4053313434123993, -0.7047193646430969, 0.2276211380958557, -0.23310579359531403, -1.4642683267593384, 0.7351301312446594, -0.5463987588882446, -1.694314956665039, -0.7929778099060059, 0.19607307016849518, -0.2689991891384125, 1.1947630643844604, 0.17766238749027252, 1.3468278646469116, -1.544793725013733], [0.550861120223999, -1.6002837419509888, 0.8509880900382996, -1.5018185377120972, 1.1026782989501953, 0.7706848978996277, -0.23116283118724823, 0.28843697905540466, -1.114365816116333, -0.9726561307907104, -1.1265777349472046, -2.609712839126587, -0.6308327913284302, 0.39090731739997864, -0.6608162522315979, 0.4858781695365906, 0.13199856877326965, -0.7492061853408813, 1.1325278282165527, 1.1204599142074585, 1.4491634368896484, -1.3689510822296143, -2.078169584274292, 0.9446549415588379, 2.4169628620147705, 0.9678401350975037, 0.6693034172058105, 2.1857762336730957, 0.8922471404075623, 0.7762678861618042, 1.1544020175933838, 0.2662062644958496, -1.69580078125, -0.9875727891921997, 1.7492694854736328, -0.32912448048591614, -0.11072275042533875, -0.6520245671272278, -0.43253082036972046, 0.4281599223613739, -1.1095664501190186, 1.1899797916412354, -0.2997084856033325, -1.9625396728515625, 0.8857249617576599, 0.061536502093076706, -0.49126356840133667, -1.626128911972046, 0.6406484246253967, -1.0793458223342896, 1.671639084815979, -1.7053298950195312, -0.723975658416748, -1.5188778638839722, 1.7840615510940552, -0.29203158617019653, -0.5552130341529846, 0.5077995657920837, 0.2942526042461395, -0.7564875483512878, -1.664258599281311, 1.0242376327514648, -0.21377350389957428, -0.5275185108184814, 0.4759398102760315, 0.20695088803768158, -0.8001552820205688, -0.7672715783119202, -1.3540325164794922, 0.45792558789253235, -1.5143187046051025, -1.7767304182052612, -0.7056446671485901, 1.1215718984603882, 0.5466128587722778, -0.09006469696760178, -1.885412335395813, 0.6303218603134155, 0.7030019760131836, 0.15696583688259125, 0.18571199476718903, -1.269208550453186, 0.38059231638908386, -0.9301297664642334, -0.9188371896743774, 2.382810592651367, 1.0432997941970825, -0.3148922920227051, -0.9238382577896118, 0.6543007493019104, -0.5658558011054993, 1.948091745376587, 0.6496879458427429, -0.35249483585357666, -0.6878600120544434, 1.4369536638259888, -0.23359860479831696, 0.03489214926958084, 0.5591263771057129, 1.3019640445709229, -0.055531568825244904, -0.8215305209159851, -0.41141650080680847, 0.032810937613248825, -0.007857056334614754, -0.7786767482757568, -2.415652275085449, 1.187583088874817, 1.1341553926467896, 0.22026623785495758, -1.6418819427490234, 0.7502533793449402, 0.6436347365379333, 0.09139896184206009, 0.45286011695861816, -0.5800750255584717, 1.5381649732589722, -0.0029510040767490864, -0.6134482622146606, -0.276305228471756, -0.3066360056400299, 0.2286471575498581, -0.030357759445905685, 0.8872854113578796, -0.6917077302932739, -0.5274263620376587, -0.1609947383403778, 0.1807178109884262], [0.7248056530952454, -0.5456179976463318, -0.22345399856567383, 1.3912166357040405, 0.8419362306594849, -0.9963493347167969, 0.43896934390068054, 2.3026435375213623, 0.5490953326225281, 1.1601625680923462, -1.0078235864639282, 0.5919238924980164, -0.16736888885498047, 0.25623300671577454, -0.10893090069293976, -0.7785311937332153, 0.5755234360694885, -1.3319131135940552, 0.10245367139577866, 0.7410807609558105, -0.7143188714981079, -0.389880508184433, -0.6998820900917053, -1.129866123199463, 1.7011457681655884, 0.6330628395080566, 0.5350289940834045, 0.5394552946090698, -1.2620971202850342, -0.7728578448295593, -1.719545841217041, 0.15497443079948425, 0.1398739069700241, -0.04884321615099907, 1.5088845491409302, 0.4104461073875427, 1.1480299234390259, 2.4608469009399414, -0.4876594543457031, -1.6340867280960083, -0.03094911016523838, 0.6574798226356506, -0.3153113126754761, -0.4346050024032593, -0.985179603099823, -1.4298995733261108, -1.3590366840362549, -0.21032387018203735, -0.6457189917564392, 0.9786885380744934, -1.6548210382461548, 1.043691635131836, -0.3256223797798157, -1.1443549394607544, -1.116874098777771, -1.3305728435516357, -2.3743722438812256, -0.06639131158590317, -0.2455197423696518, 1.3084677457809448, -1.6012500524520874, -0.9339758157730103, -2.8097877502441406, 0.10385604202747345, 1.1640739440917969, -0.3284395635128021, -0.21627676486968994, 0.15208002924919128, 0.10233495384454727, -0.48516494035720825, -0.8381925821304321, -0.8333516120910645, -0.6044234037399292, -1.0616483688354492, -0.6041028499603271, -0.9794935584068298, 1.1345610618591309, 1.5001113414764404, 0.5647119879722595, -0.2863600254058838, -0.45706743001937866, -0.8642526268959045, -0.9769030213356018, 0.3153812289237976, -0.23159906268119812, 0.7648696303367615, 0.4859463572502136, -1.985399603843689, 1.184162974357605, 1.576770544052124, -2.347191095352173, -0.864935040473938, -0.41968652606010437, 0.4751378297805786, -0.34460142254829407, 0.5118546485900879, 1.071670413017273, -0.14005498588085175, 0.6771440505981445, 0.36734652519226074, -0.7376047372817993, 0.316240519285202, 0.28860530257225037, -1.7131764888763428, 0.613123893737793, -0.13591048121452332, 0.17501342296600342, -0.5812423229217529, -0.40856054425239563, -1.024337649345398, 1.2467725276947021, 0.14823845028877258, 0.08765818178653717, -1.6284034252166748, 0.03458718582987785, -0.11585395783185959, -1.2175847291946411, 0.46361902356147766, -1.1325637102127075, -0.5548372268676758, 1.3747813701629639, 0.6213403344154358, 0.1669289618730545, -1.61506986618042, -2.1281590461730957, 0.13810408115386963, -0.5168895721435547, 0.7754406332969666], [-0.3369387090206146, 1.0584132671356201, -1.7154594659805298, 0.5022520422935486, 1.2828152179718018, 0.8083357810974121, -0.6187667846679688, 1.1217414140701294, 0.07237661629915237, -0.7009553909301758, 0.366439551115036, 1.6417932510375977, -1.3400369882583618, 1.097309947013855, -0.12957866489887238, -1.084322452545166, -0.2232362926006317, -1.6625468730926514, 0.2326069325208664, 0.14168551564216614, -0.08169714361429214, -0.43660929799079895, 0.6627531051635742, -0.16719244420528412, 1.3453999757766724, -0.41082966327667236, 0.8699740171432495, 0.5348884463310242, -0.1597922146320343, -1.1210756301879883, -0.5872906446456909, 0.37765172123908997, -0.18247582018375397, -0.016657283529639244, -0.8324187397956848, 1.5891560316085815, 0.880560040473938, -0.7337680459022522, 0.81191086769104, -0.18782982230186462, 0.40129512548446655, 0.6791019439697266, -1.5538458824157715, -0.9715026617050171, -0.8712924122810364, -1.4547926187515259, 0.7493993043899536, -0.565666675567627, 0.023060940206050873, -0.00292418384924531, -0.18639487028121948, 0.5292009115219116, -0.009078814648091793, 0.13229331374168396, 1.4415912628173828, -0.2853788137435913, 0.5278316140174866, 0.3072870075702667, -0.08130543678998947, -1.244955062866211, -0.8608112931251526, 0.22398899495601654, -0.5313779711723328, 0.7688704133033752, -0.5876219272613525, -1.3205757141113281, 0.5268166065216064, -0.800696849822998, -0.5170577764511108, 0.08013484627008438, 0.842288076877594, -0.4913158714771271, 0.3616512715816498, 0.7362803220748901, -1.8915084600448608, 0.37826284766197205, -0.4993915259838104, 1.561466932296753, -0.8982223272323608, -0.8754684925079346, 0.9864291548728943, -0.8135777711868286, 0.7574375867843628, -0.6604925990104675, 2.735363483428955, 0.5459116697311401, 0.41071754693984985, -1.211606740951538, 0.3221908211708069, 0.6547085046768188, -0.8582555651664734, 0.15840761363506317, 0.284630686044693, 0.25945210456848145, -0.3835330605506897, -2.31074857711792, -0.11762133240699768, 0.8707194328308105, -0.26560473442077637, -0.729015052318573, 0.5353963971138, -0.1611979901790619, -0.6353667378425598, -0.14139388501644135, 0.699439287185669, -1.5191911458969116, 0.5657088160514832, 0.1777634471654892, 1.431455373764038, -1.9067833423614502, 0.17750702798366547, -1.5859116315841675, -0.19095216691493988, 0.8300084471702576, 1.1598130464553833, 1.2196838855743408, -2.0052542686462402, 1.3357654809951782, -0.5050818920135498, 0.092755027115345, 0.9437501430511475, 0.22666189074516296, -0.14206768572330475, 1.5694401264190674, 1.416397213935852, 0.5637969970703125, -0.07400213927030563, -0.4619148373603821], [-0.041365232318639755, -0.34301450848579407, 0.05978773534297943, -0.2632668614387512, 0.3436969816684723, -0.6993598341941833, -1.2570397853851318, 1.2498793601989746, 0.49189889430999756, -0.04277767986059189, -0.4582500755786896, 0.6099244952201843, -1.9233217239379883, 0.5544061064720154, -1.3655104637145996, 0.2316139042377472, -0.8585172295570374, -1.2703770399093628, 0.8868455290794373, 0.39926570653915405, -2.1977343559265137, -1.7887576818466187, -0.6845226883888245, -2.0002896785736084, -0.11052866280078888, 0.2889872193336487, 0.33164894580841064, -0.041798584163188934, 1.467753291130066, -0.5601311326026917, 0.03366474062204361, 0.6874080300331116, -0.6756943464279175, 0.8093962669372559, -1.8014742136001587, -1.3053898811340332, -0.021749136969447136, -1.113850474357605, -0.7713532447814941, 0.31963813304901123, 0.7120182514190674, -0.39192163944244385, 0.49960631132125854, 0.09063529968261719, -0.23077307641506195, 1.2942990064620972, -0.5514940023422241, -0.19332599639892578, 0.491067111492157, -1.6401420831680298, 0.8277313113212585, 1.9647384881973267, -0.5448150634765625, 0.19611793756484985, 0.38264209032058716, 1.628449559211731, -2.0200297832489014, -0.053761668503284454, -0.20515522360801697, -0.47152212262153625, 0.14981135725975037, 0.0806911438703537, -0.07003474980592728, 0.2845921516418457, 0.7243254780769348, 0.589436411857605, 0.15964804589748383, 0.3278924226760864, -1.5448188781738281, 1.6360169649124146, -1.0054571628570557, -0.522983968257904, 0.22839397192001343, -0.4944295287132263, -0.3531050682067871, -0.35051873326301575, -0.7068252563476562, -1.9567080736160278, 0.3571159243583679, 1.334775447845459, -1.3190149068832397, -0.8284181952476501, -1.228769302368164, -0.23340100049972534, 1.1950467824935913, 0.6913885474205017, 0.9349039793014526, -0.3544975817203522, 0.1143743023276329, 0.2353680431842804, 0.4321447014808655, -1.0688319206237793, -2.526703119277954, -1.4897446632385254, -0.15614621341228485, -0.6385815143585205, -0.28328242897987366, 0.2424265742301941, 0.27362242341041565, -0.2757747769355774, 1.321567177772522, -1.402274250984192, 0.3808990716934204, -0.049970731139183044, -1.546664834022522, 2.3200559616088867, 0.18246695399284363, 0.17459554970264435, -0.9571930766105652, 0.1953653246164322, -0.533001720905304, -0.7675344347953796, 0.601388156414032, 0.5823842287063599, 1.4690111875534058, -0.21567854285240173, -0.03519187122583389, -2.954686403274536, -0.23701463639736176, 0.5440512299537659, -0.4409295320510864, -0.7267302870750427, -1.3266572952270508, 0.762231171131134, 0.1551067978143692, -2.369915246963501, 2.318450927734375, 0.4435485005378723], [-2.5867979526519775, 2.165710210800171, -1.2611701488494873, -0.5774458050727844, -2.4617629051208496, -0.3285483717918396, 0.3378662168979645, 1.0729577541351318, 0.6885895729064941, -0.12746813893318176, -0.05257254093885422, -0.27483364939689636, 0.40964600443840027, -0.6736696362495422, -0.6665399074554443, -1.20616614818573, 0.8981523513793945, 1.120139718055725, 0.5294146537780762, 0.000978716998361051, -0.12071641534566879, 1.0522263050079346, -1.1704046726226807, -0.07418916374444962, -1.1962207555770874, 0.3763044774532318, 1.159757375717163, -0.5529072880744934, 1.8196285963058472, 0.11182938516139984, -0.23480810225009918, -0.6624782681465149, -1.8080042600631714, -0.4828922152519226, 1.1548882722854614, 0.3822751045227051, -0.06876382976770401, -0.26765480637550354, 0.329710453748703, -2.0977895259857178, -0.3074774742126465, -0.9984142780303955, -0.8284130692481995, -2.2910540103912354, 0.8522553443908691, -0.060591138899326324, 0.31773313879966736, -0.7381564974784851, -0.05000689998269081, 0.4995623528957367, -1.5634323358535767, 2.1305649280548096, 0.1254928559064865, 1.6390392780303955, 0.6724602580070496, 2.101118564605713, 0.34634602069854736, 0.4575996398925781, 0.7893637418746948, 0.09576676785945892, -0.6492459774017334, -1.2864632606506348, 1.3481334447860718, -0.9081732630729675, 0.5240689516067505, 0.3150476813316345, -1.478954553604126, 1.8383245468139648, -1.0079559087753296, 1.2550387382507324, 0.32550862431526184, 0.34564703702926636, -0.31992337107658386, -0.22225068509578705, -1.311745047569275, 0.18088671565055847, -1.4727692604064941, 0.8224987983703613, -1.123644471168518, -0.7238965630531311, -0.28768429160118103, 0.08532809466123581, 0.4474952816963196, -0.18262910842895508, -1.3802894353866577, 0.5426697731018066, -0.3309212625026703, 0.45847901701927185, 0.11625099927186966, -0.23381859064102173, -0.7874758243560791, 0.5832914113998413, 0.16951143741607666, 0.5251147747039795, -0.3723224401473999, 0.3968941271305084, -1.1385514736175537, -1.1537059545516968, 0.4358040392398834, -0.9326266646385193, 0.35822755098342896, -1.1009544134140015, -0.763785719871521, 0.44343164563179016, -0.40342164039611816, -0.6049925684928894, 0.17378468811511993, -0.632345974445343, -1.034940481185913, 0.3283335566520691, 0.3544081449508667, -0.5932481288909912, 2.4199230670928955, -1.0172580480575562, 0.22941362857818604, 0.9241234660148621, -1.144943356513977, 0.2586393356323242, 0.2021239846944809, 0.8585061430931091, 0.05562112480401993, -1.338793396949768, -0.2420685738325119, 0.5217000842094421, -0.5568668246269226, 0.9157915711402893, 0.5481014251708984, -0.3236669600009918], [-0.10354503244161606, 0.29412946105003357, -0.18434253334999084, -0.12347371876239777, 0.07668006420135498, 0.4514695107936859, -0.2740101218223572, -0.8587133884429932, -0.16937820613384247, 0.7193369269371033, -1.0725939273834229, -0.21344570815563202, 0.008405872620642185, -0.008873985148966312, 0.522941529750824, -0.22559812664985657, -0.42232486605644226, 1.5030781030654907, 1.3066271543502808, 0.34890130162239075, -0.135647714138031, 2.179981231689453, -1.5080125331878662, -0.08538171648979187, -0.15543609857559204, -0.6885639429092407, 0.9751752018928528, 0.6099752187728882, 0.9930353760719299, 1.2157288789749146, -0.33666881918907166, 1.1469756364822388, -0.6782821416854858, -0.6534510254859924, -0.9906182289123535, 2.6741230487823486, -0.9211770296096802, 0.8917542695999146, -0.8442416191101074, -0.5823677778244019, -0.1462264508008957, -0.8798708915710449, 0.22258363664150238, 0.5978925824165344, 0.43940040469169617, 1.65609610080719, -0.6169466972351074, -1.24032461643219, 2.1005654335021973, -0.360302597284317, -1.005704402923584, -0.42439907789230347, -0.5293657779693604, 0.33513420820236206, 1.2747310400009155, 0.5318343639373779, 1.3376412391662598, 0.9903036952018738, 0.4243336617946625, -0.964992344379425, -1.2894505262374878, 1.1577237844467163, 0.8448042869567871, 0.9683451056480408, -0.7047789692878723, 0.55843585729599, 0.5041615962982178, 0.8341549038887024, 0.9921752214431763, 0.4104660749435425, 0.399111807346344, 1.5223522186279297, -0.4609954357147217, -0.6622853875160217, 0.8873960971832275, 0.4645293653011322, -0.5628014206886292, -2.1618502140045166, 0.8889533877372742, 0.05323706194758415, -0.5069707036018372, 1.2576910257339478, 0.21073170006275177, 1.257334589958191, 1.311318278312683, 0.367707222700119, 0.3515179753303528, 0.9116812944412231, -0.4953472316265106, -0.03200880065560341, 0.9111045002937317, -0.5480480194091797, -1.862838864326477, -0.6453122496604919, 0.6871121525764465, -0.5104718208312988, -0.01574043743312359, 0.18757525086402893, 0.5583573579788208, 0.238168403506279, 2.3829338550567627, -0.42932289838790894, -0.04452865570783615, 0.7241389155387878, -0.9709708094596863, 1.1322102546691895, 1.9046066999435425, 0.3788287341594696, 2.822603940963745, -0.47516921162605286, 0.4597824215888977, -0.037461623549461365, 0.9829756617546082, -0.6192866563796997, -0.23161593079566956, -0.8936046361923218, 0.5358874201774597, 0.41874414682388306, -0.6483897566795349, 1.8005635738372803, -0.4689296782016754, -0.10383392125368118, -2.084151029586792, 0.9332906603813171, -0.30967435240745544, 0.8204983472824097, -0.777264416217804, 1.0275475978851318], [-0.9629566073417664, -1.2487508058547974, 1.0756759643554688, 0.20486462116241455, -0.6911641359329224, -0.12983635067939758, 1.35383939743042, -0.28377410769462585, 2.340914726257324, 0.026747960597276688, 0.9357924461364746, -2.42299222946167, -0.07078344374895096, 0.0383017435669899, -2.6237165927886963, -0.33382898569107056, 0.7593532800674438, -0.4863249361515045, -1.6211135387420654, 1.5575753450393677, 0.9681898951530457, 0.05651938542723656, -1.0759074687957764, 0.8797066807746887, 1.1213120222091675, 0.26187992095947266, -0.05419964715838432, -0.6312747001647949, -0.8302932977676392, -1.9149733781814575, -0.35160914063453674, 0.12560096383094788, -0.11089321225881577, -1.2267658710479736, 0.5528278350830078, -0.2601185441017151, -1.4126065969467163, 0.5346100330352783, -0.6782037019729614, -0.2015266716480255, -0.15073591470718384, -1.7910747528076172, -0.2447267770767212, -1.2554142475128174, 0.9635255932807922, 0.6202710866928101, 0.5043664574623108, 0.16962043941020966, -1.0673561096191406, 1.2443934679031372, -0.8114075660705566, 0.13381843268871307, 1.35172438621521, -0.5466219782829285, 0.6925360560417175, 0.3143152892589569, 0.9583421349525452, -0.9638684988021851, -0.9189579486846924, -1.506608009338379, -0.7009797692298889, -0.51751708984375, -0.42766690254211426, -0.3721121847629547, 0.40600714087486267, 0.3968900442123413, -0.6340023875236511, -1.215836763381958, 0.6494617462158203, 1.2121855020523071, -0.5698261857032776, -0.6212264895439148, 0.9246649742126465, -1.1444371938705444, 0.9839890003204346, -1.8316706418991089, 1.8985098600387573, -0.4044269621372223, 0.4184136390686035, 0.9566542506217957, 0.2977762818336487, -0.2204405665397644, -0.014506793580949306, 0.4065355062484741, 0.10340700298547745, -0.281265527009964, -0.12026279419660568, -0.15509182214736938, 0.8099395632743835, -0.3427794873714447, -0.8188617825508118, 0.6014295220375061, 1.240767478942871, -0.7226454615592957, -0.5206502676010132, -0.774203896522522, -1.0653184652328491, -0.5348812341690063, -0.9074638485908508, 2.6687841415405273, 1.3595014810562134, 2.6338603496551514, -0.020444653928279877, 0.32229045033454895, -0.736108124256134, 0.13332097232341766, -1.0461846590042114, 0.5258182287216187, 1.7271206378936768, 0.6716241836547852, 0.49832719564437866, -0.4620799720287323, -0.2908303439617157, -0.2914009094238281, 0.28342241048812866, -0.7617587447166443, 1.443199872970581, 0.5098914504051208, 0.4028628468513489, -1.496402621269226, 1.1307495832443237, -0.4831899404525757, 1.2466524839401245, -0.2907028794288635, 2.0505149364471436, -0.5640110373497009, 0.21856854856014252, 0.7084842920303345], [0.3210437595844269, 0.2116064727306366, 0.8160446286201477, 0.5936971306800842, 0.045389145612716675, -0.132359579205513, -0.33325886726379395, -0.5441915392875671, 1.3342279195785522, -0.9352295994758606, -1.029632806777954, -0.02418399602174759, 0.3396925926208496, 0.7745457887649536, -0.17004218697547913, 0.6638040542602539, -0.12444635480642319, -0.8060743808746338, -0.024258315563201904, 2.0747454166412354, -1.4130345582962036, 0.539874255657196, 0.7272090911865234, -0.8240043520927429, 0.10970252752304077, -0.02711065299808979, 0.429400771856308, -0.4375651776790619, -0.18057163059711456, -0.3868454694747925, -0.02814040146768093, 0.8189793825149536, -1.3399542570114136, 0.281333863735199, 0.21791872382164001, 0.9361302256584167, -1.0449374914169312, -0.5748921632766724, -0.2172294706106186, 1.357716679573059, 0.32613515853881836, 1.4777915477752686, 0.034553200006484985, -1.5322567224502563, 1.5016562938690186, -0.755557119846344, 2.0094335079193115, 0.3465586006641388, 0.21720226109027863, -0.3810451924800873, 0.0466747023165226, -0.39501941204071045, 1.7659077644348145, 0.07609134912490845, 0.4093439280986786, 0.18269532918930054, -0.0001166680158348754, 0.19858595728874207, 1.9132630825042725, 1.5754692554473877, -0.4716523289680481, 1.4509447813034058, 0.31100907921791077, 0.908724844455719, 0.00902000442147255, 0.814871072769165, 0.35417890548706055, 0.25626063346862793, 0.17047318816184998, 0.4029124677181244, -1.3555879592895508, 0.865553081035614, 1.153219223022461, -0.2553037405014038, 1.688918948173523, -0.9589047431945801, 0.5877072811126709, 0.5960062146186829, 0.6900021433830261, -0.00842125341296196, -0.05201224237680435, -1.8448776006698608, -0.5504817962646484, -0.8686283826828003, 1.4624446630477905, -1.7912986278533936, 0.7565803527832031, 1.8516156673431396, -0.2397928088903427, 0.06477654725313187, 1.4879579544067383, 0.06324036419391632, 1.0491652488708496, 0.26561710238456726, 0.8544726967811584, 0.9677030444145203, -0.31439122557640076, 2.4419710636138916, -0.7292569279670715, -1.0239230394363403, -1.540238380432129, -0.9334604144096375, -0.6106712818145752, 1.3448785543441772, -0.8582161068916321, 0.05511295795440674, -0.03538913279771805, -0.6856025457382202, -1.621323585510254, -1.577170729637146, 1.4259650707244873, 0.7729493379592896, -1.0478973388671875, -0.34506741166114807, 1.1725404262542725, 1.0801056623458862, -0.7085813879966736, -1.364458441734314, 1.5703456401824951, -1.814133644104004, 0.7547795176506042, 1.4749826192855835, -1.4047961235046387, -0.9102084040641785, -1.673475742340088, -0.5997888445854187, -0.43505293130874634, -0.2913374602794647], [-0.9806604981422424, -0.7032353281974792, -0.8611248731613159, 1.142124891281128, -0.36264944076538086, -1.1649922132492065, -0.6742696166038513, -0.9369685649871826, 0.981367826461792, -0.500686764717102, -1.4199079275131226, 0.8502592444419861, -1.7228649854660034, -0.14159870147705078, -0.1952272653579712, 0.4218319356441498, 0.21460483968257904, -1.7503468990325928, -0.016873663291335106, 1.68923020362854, -1.3909236192703247, 0.5774984359741211, 1.30491304397583, -1.367021918296814, -0.0960884839296341, -1.3674873113632202, 0.21870848536491394, -1.4753445386886597, -0.19954663515090942, 2.4270873069763184, -0.11672750115394592, -0.11275875568389893, -0.15888693928718567, -0.11327949911355972, 1.1819560527801514, 1.638641357421875, -0.13652493059635162, 1.7634555101394653, 0.6531078219413757, -0.42519739270210266, 1.0972775220870972, -0.52412348985672, -0.5925520658493042, 0.45715129375457764, 1.183045506477356, -0.482876181602478, -0.6052656173706055, 0.35744377970695496, 0.35144054889678955, -1.0564552545547485, 1.2697184085845947, 0.7988793253898621, -0.21751952171325684, 1.1384576559066772, 0.35883456468582153, -1.2922757863998413, 1.128826379776001, 0.841160774230957, -0.5282521843910217, 0.3267502188682556, 1.2119742631912231, 1.2612364292144775, -0.299893319606781, 2.1736881732940674, 0.8842195272445679, 1.8255698680877686, -1.2733134031295776, -0.2806989848613739, 0.15311717987060547, 0.27713868021965027, 1.4471131563186646, 0.5793464779853821, -0.03397383168339729, 1.327575922012329, -0.0890255942940712, 0.9039316177368164, -0.26382166147232056, -0.4866141378879547, 1.4905295372009277, -0.9924526214599609, 0.5845121145248413, 0.9714452028274536, 0.5691196918487549, -0.32734885811805725, -1.0480096340179443, 0.07983803749084473, 0.5700657367706299, 0.5010905861854553, -0.35993948578834534, 0.17533034086227417, -1.6950483322143555, -0.0475224107503891, -1.1853841543197632, 0.257685124874115, 0.5542395114898682, -0.5032546520233154, 1.258408546447754, 0.1588549017906189, 0.4089103937149048, 0.10262523591518402, 0.2462090402841568, -0.8845040798187256, -0.20255056023597717, 0.5238880515098572, 0.3716188967227936, -0.1409902274608612, -0.5088094472885132, 2.1061525344848633, -1.1752599477767944, -0.43715107440948486, -0.21768724918365479, -0.4700923562049866, 1.206098198890686, -1.2507736682891846, -1.2328290939331055, -1.321923851966858, -1.1952874660491943, 0.5432929396629333, 1.277353048324585, 0.3784726858139038, -1.376186490058899, -1.7119619846343994, 0.33929163217544556, -0.9260379076004028, -0.46680372953414917, -1.9247996807098389, -0.5146417617797852, -0.5768449902534485], [-1.824592113494873, -1.0784357786178589, 1.4714020490646362, 1.1698108911514282, 0.5902985334396362, 1.4920202493667603, 0.4847268760204315, -1.090524673461914, -0.17817552387714386, -0.13206055760383606, 0.23423781991004944, 1.0227144956588745, -0.7324624061584473, 0.5521368384361267, -0.6669858694076538, 1.4218101501464844, 0.831485390663147, -0.7110017538070679, -0.4201086759567261, -1.4592912197113037, -0.4265805780887604, -0.16464056074619293, 0.14387205243110657, -0.297247976064682, -2.317526340484619, -0.6590131521224976, -0.505096435546875, -1.2007759809494019, -0.16552592813968658, 1.7940738201141357, -0.26517006754875183, 0.35469475388526917, -2.2325189113616943, 0.15682464838027954, 0.26048871874809265, 0.34862837195396423, -0.030436333268880844, 0.7405903935432434, 0.4541916251182556, 0.20292623341083527, -0.9934574961662292, 0.19182445108890533, 2.8778607845306396, -0.9849515557289124, -1.211347222328186, 1.0331875085830688, -0.5780856013298035, 0.6648446321487427, -0.11069007962942123, -0.3881417512893677, 0.27960434556007385, 0.2875257432460785, 0.40396639704704285, 0.5693721771240234, -0.9868724346160889, -1.583816409111023, 0.11044012010097504, 1.0675263404846191, 1.1376782655715942, -1.9806307554244995, 0.3410428464412689, 0.7093093991279602, 0.6015480756759644, 1.1290069818496704, 0.4331319034099579, -0.4372483193874359, -1.0893213748931885, 0.49821409583091736, 0.9310609102249146, -0.023894859477877617, -1.1124275922775269, 0.7427234649658203, 0.7634314298629761, -1.8848774433135986, -0.105802021920681, 1.8825569152832031, -1.1546555757522583, -0.6391404271125793, -2.9793379306793213, -0.02578870952129364, 1.9190301895141602, 0.7607088685035706, -0.6439675688743591, 0.0345151461660862, 2.051497459411621, 0.8561277985572815, 0.5663543939590454, -1.1014809608459473, -0.12230177968740463, 0.5829750299453735, 0.4135463237762451, -0.9320701956748962, 1.8791346549987793, 0.08525310456752777, -1.0915433168411255, 1.3731786012649536, 1.3564841747283936, -0.7330632209777832, -0.6766387820243835, 0.3507261574268341, 0.04885122552514076, -0.17338497936725616, 0.8632403612136841, 0.47879523038864136, -1.4257055521011353, -0.675957202911377, 2.0632400512695312, -2.4170100688934326, -1.2903708219528198, 0.4949870705604553, 0.3645097017288208, 0.21755175292491913, 0.6203315258026123, 0.28331422805786133, -0.7763705849647522, 0.20031382143497467, 1.1696887016296387, -1.3695083856582642, 1.6495120525360107, -0.44445255398750305, -0.7701156139373779, 1.3954051733016968, -0.17573095858097076, -1.1101011037826538, 0.989737868309021, -1.1640733480453491, 0.26526567339897156, -1.5291736125946045], [0.7569635510444641, 1.6744868755340576, 1.0104416608810425, -0.40064942836761475, 0.4719146490097046, 0.9801095724105835, 0.8724814057350159, -0.5484398007392883, -0.08091799914836884, -0.570275604724884, -0.8117756247520447, -0.8153510689735413, 0.0438646599650383, -0.10091952979564667, 1.919770359992981, 0.5643632411956787, 1.8952512741088867, 0.27320966124534607, 0.38976097106933594, 0.3704376518726349, -1.4901180267333984, 0.9290827512741089, 0.5386424660682678, -2.6337380409240723, -0.22097715735435486, 2.026590585708618, -0.6734390258789062, -0.23185239732265472, 1.0497257709503174, 0.008994543924927711, 0.7243160009384155, 1.042382001876831, 0.7431557178497314, -1.9197531938552856, 1.375734806060791, 0.5154474973678589, -0.5552363395690918, -0.9265751838684082, -0.24519729614257812, 0.11424969881772995, -1.311203956604004, 1.5816316604614258, -0.9002759456634521, -0.14647097885608673, -0.5978180766105652, -0.45890191197395325, -0.591010332107544, -0.38115987181663513, 0.5870726108551025, -0.08276399970054626, -0.052097491919994354, 0.4160902500152588, 0.7537116408348083, -1.4522498846054077, 0.4355459213256836, -0.0024808694142848253, 0.3238249719142914, 0.8222692608833313, 0.49990010261535645, -0.8554880619049072, 1.0055699348449707, -0.2695976793766022, -0.8177878260612488, 0.5132935047149658, 0.11161833256483078, -1.761256217956543, -0.5422370433807373, 0.9026356935501099, 0.3473820090293884, -0.7617737054824829, -0.9343904256820679, -0.06841355562210083, -1.3709609508514404, 0.836746096611023, -0.8619173765182495, 0.2527461051940918, -0.6321061849594116, 0.40684548020362854, -1.778035283088684, 1.8202091455459595, -0.23571966588497162, 1.4718947410583496, -1.939184308052063, -2.50842022895813, 0.6057426333427429, 0.9897405505180359, 1.299870252609253, -1.1382805109024048, 1.4574342966079712, 1.4393401145935059, 1.3335641622543335, 0.5584449768066406, 0.9529978632926941, -0.33632105588912964, 1.4529982805252075, 0.9490828514099121, -0.19781935214996338, -0.3272883892059326, -0.7688826322555542, -0.10955631732940674, 0.9575356841087341, 1.4373455047607422, -1.1794918775558472, -0.06959175318479538, 0.506388247013092, -0.24068139493465424, -0.8776005506515503, -0.9018012285232544, 1.0598193407058716, -0.17975196242332458, 0.8785749673843384, -0.7876264452934265, 0.5665019750595093, 1.8528441190719604, -1.2370227575302124, 0.35067665576934814, -0.2347724586725235, -0.9874962568283081, -1.9801054000854492, 1.226850986480713, -1.5937309265136719, 0.5589422583580017, 0.5536475777626038, 0.6047728061676025, 0.37849730253219604, -1.3362457752227783, -0.22610992193222046, 1.686111569404602], [-0.8296763300895691, 1.299092173576355, 0.023450519889593124, 2.2684924602508545, -1.8208844661712646, 1.321446418762207, -1.0851025581359863, 2.412292003631592, 1.6029059886932373, -0.9235089421272278, 0.3443416953086853, 1.4007606506347656, 0.9001098871231079, -0.4338648319244385, 0.426104873418808, 0.2044752687215805, -0.8640519380569458, 0.8994717597961426, 0.21017968654632568, -2.702364444732666, 0.48007693886756897, -0.2256038933992386, -0.3047283887863159, -0.09248249977827072, 1.0460435152053833, 1.535447359085083, 0.3550828695297241, 0.7568509578704834, -0.3026500642299652, -0.8962259888648987, 1.6905542612075806, -1.0963395833969116, 1.5996490716934204, -0.8650141358375549, 0.5527515411376953, -1.6913059949874878, 0.32096582651138306, -0.33987152576446533, 0.9517778158187866, 0.3477936089038849, 1.5583134889602661, 0.5975027084350586, -0.09466376900672913, -0.2620641589164734, 0.056092388927936554, -1.118801474571228, -0.4721527099609375, -0.6366707682609558, -1.7311275005340576, 0.5162289142608643, 1.2699027061462402, -0.2025098353624344, -0.560046911239624, -0.09499455988407135, -1.1471325159072876, -1.578073263168335, -0.5088286399841309, -0.6901755928993225, 0.22925278544425964, -0.0357673317193985, 0.8800845742225647, -0.9513397812843323, -0.818599283695221, -0.9123839735984802, -0.32335329055786133, 0.04842546954751015, 1.4620287418365479, -1.0002751350402832, -0.5134477615356445, 1.473652958869934, 0.528596043586731, 1.3380475044250488, -0.7016807198524475, -0.49074044823646545, 0.5699030160903931, -2.2792437076568604, -0.16411210596561432, -0.5369028449058533, -1.3049578666687012, -0.22942601144313812, 0.9660118222236633, 0.39459455013275146, -0.09666141122579575, 0.5954211354255676, -0.5530166029930115, -0.5872302055358887, -0.5150029063224792, -2.24981951713562, 0.391353040933609, 1.1574994325637817, -0.8869112133979797, 0.7042246460914612, -1.2006078958511353, 1.081562876701355, -1.423938274383545, 0.5982099771499634, -0.6852119565010071, 0.9638488292694092, -1.063525915145874, 0.7822751402854919, 0.48454728722572327, 1.705183982849121, 0.2010224312543869, 0.12091057002544403, 0.3948741555213928, -0.10308749228715897, 1.6527167558670044, 0.3792537748813629, 1.019055724143982, -0.9768548011779785, 1.358742117881775, 0.5829475522041321, -1.2518277168273926, -0.19746321439743042, 0.07742343097925186, 0.34282657504081726, 1.3387268781661987, -0.4080294966697693, -0.5939579010009766, -0.20532841980457306, 0.12440922856330872, 0.24435310065746307, 0.910283625125885, -1.8836323022842407, 1.3609881401062012, -0.42415741086006165, -0.9066532850265503, -0.46436259150505066], [-0.5491686463356018, 1.2485069036483765, 1.1763005256652832, -1.663278341293335, -0.09951432794332504, 1.0302355289459229, -0.3873363435268402, -0.30462485551834106, -1.8121278285980225, 1.0544352531433105, 1.4778467416763306, 0.6788762211799622, 1.2334901094436646, -1.1307337284088135, -0.31426793336868286, 0.6514925360679626, -1.841810941696167, -0.07880917936563492, -0.0367463082075119, 0.7068172097206116, -1.3639646768569946, 0.08242647349834442, 1.311833143234253, -1.2237493991851807, 0.3302477300167084, -0.11389598995447159, -0.526603639125824, -0.6664672493934631, 1.2022349834442139, -0.23047009110450745, -0.4449574947357178, -2.0824315547943115, -0.6660215854644775, -0.14212319254875183, -0.22314049303531647, -0.09589216858148575, -1.1804943084716797, -0.9490644335746765, -1.4962153434753418, -0.5937564969062805, -0.2616441547870636, 0.46602991223335266, -0.4643418490886688, 0.8147707581520081, 0.7385210394859314, -1.1443486213684082, 0.3549870550632477, -0.395283043384552, -0.2076861709356308, 0.9077290892601013, -1.4950335025787354, -1.5051671266555786, -1.8425240516662598, -0.514955997467041, -1.161754846572876, -0.8123315572738647, 0.30297666788101196, 1.106826901435852, 0.19849999248981476, 1.680418848991394, 0.5365053415298462, -0.10917573422193527, -0.45456835627555847, 0.6944599151611328, 1.1764851808547974, -0.6453471183776855, -0.33656349778175354, 1.722755789756775, -2.3116376399993896, -0.6627458930015564, -0.18244194984436035, -0.2694690227508545, -0.7495408654212952, -0.7055478096008301, -0.6171051263809204, 0.2985900640487671, 0.3411477208137512, 0.37455862760543823, 1.0364149808883667, -0.6318444013595581, 0.7708097100257874, 0.8738576769828796, 0.6613374352455139, 0.7536190748214722, 0.7076880931854248, 1.53812575340271, 2.685128927230835, 2.4664320945739746, 1.05569326877594, 0.41915905475616455, 0.9543363451957703, 0.1405208259820938, 1.3574961423873901, 1.9276736974716187, 0.21949626505374908, -0.7369973063468933, -1.1829183101654053, 0.22616879642009735, 0.08053962886333466, 0.0005576914991252124, 0.3403608202934265, -0.7134871482849121, 1.4966133832931519, 0.1783970445394516, -0.6353947520256042, 1.597917914390564, -0.5102381706237793, -0.5802053809165955, -0.22277700901031494, 0.18953128159046173, 0.3816494643688202, 2.498485565185547, -0.24000878632068634, 0.9368844032287598, -0.4304213225841522, -0.29889774322509766, 1.270642876625061, -0.793293297290802, -1.3069875240325928, 1.897606372833252, 0.8880518674850464, -1.6958467960357666, -2.1862287521362305, -0.1142747551202774, -1.1170344352722168, -1.1634587049484253, 0.03964382782578468, 0.8266208171844482], [0.1703444868326187, -1.0421407222747803, 1.2475143671035767, -2.0010457038879395, 0.530653715133667, -0.8644345998764038, -0.5831479430198669, 0.39772841334342957, 1.3085217475891113, 0.016055790707468987, -0.2140871286392212, -0.15505573153495789, 0.587440013885498, -1.1059976816177368, 0.5394435524940491, -2.8764963150024414, 2.2405123710632324, -1.6364375352859497, -0.8604456782341003, 0.741022527217865, 1.6631004810333252, -1.21802818775177, 1.7764172554016113, 0.8098073601722717, 0.5007696151733398, 1.0546643733978271, -0.7276665568351746, -1.707724690437317, -0.6217606067657471, -0.7737146019935608, 0.2070731222629547, -0.28713056445121765, -0.9741770029067993, -0.10289083421230316, -0.8496991395950317, -0.2719458341598511, 0.12492981553077698, 0.03429212421178818, 1.001533031463623, -0.7929449677467346, 0.9789808988571167, -0.19052381813526154, -0.9051229953765869, -0.07551658898591995, -0.4066540002822876, 0.6658419370651245, -1.8172560930252075, -0.2360650897026062, 0.45441749691963196, 0.07270772755146027, -0.018855946138501167, -0.09245336800813675, 0.08667929470539093, -0.8907976746559143, 0.5817063450813293, -0.07495324313640594, -0.5974228382110596, 1.6773790121078491, 0.6496627926826477, -0.24921156466007233, -0.41990575194358826, 0.2351502925157547, -0.5315492749214172, -0.9399434924125671, 2.0066721439361572, 0.2178904265165329, 0.08140698075294495, -0.18166813254356384, 0.8618612289428711, 0.364918053150177, -0.23125888407230377, -0.6418948769569397, -1.0103942155838013, 0.9614297747612, 0.4821423888206482, 0.9489249587059021, 1.5380281209945679, 2.059246063232422, -0.2166927307844162, -2.08966064453125, 0.5235106348991394, -0.5068898797035217, 1.411775827407837, 0.9802467823028564, -0.6223383545875549, 0.3455028831958771, -0.41050413250923157, -0.4817357659339905, -1.1524696350097656, -1.6266785860061646, 0.3028411865234375, 0.010144662111997604, -0.8548511862754822, 0.42205363512039185, -2.332242488861084, 3.7106635570526123, -0.2379009872674942, 0.3339063823223114, 0.9394364356994629, 0.6404125094413757, 2.0142898559570312, 0.3471863865852356, -0.6864508986473083, -0.3705769181251526, -0.13175058364868164, 1.3397417068481445, 0.4963512420654297, 0.33304205536842346, 2.0069942474365234, -0.1523188054561615, -1.3957469463348389, -0.6972726583480835, -0.27434036135673523, -0.11518184095621109, -0.7021275162696838, -1.7242083549499512, -0.7357265949249268, 0.2568766474723816, 1.029660940170288, 0.49581554532051086, -0.7808252573013306, 0.713943600654602, 1.3400863409042358, 1.240054726600647, -0.14533056318759918, 1.2271345853805542, 0.14246800541877747, 0.46787330508232117], [-0.4878605604171753, -1.7439287900924683, -0.5720717310905457, 0.017997711896896362, -0.30679699778556824, -0.07599279284477234, 0.027347952127456665, 0.5930087566375732, -1.41312575340271, -1.1964045763015747, 0.9007646441459656, 1.072189450263977, -0.3549748361110687, -0.7820167541503906, 0.8639442920684814, -1.3953285217285156, -0.5088895559310913, 0.5501572489738464, 0.2391640692949295, -1.305361270904541, 1.0930808782577515, 0.3867863714694977, -0.0061734868213534355, -0.4851137399673462, 0.027169523760676384, 0.657261073589325, -2.2887589931488037, 1.1945455074310303, -0.8481540083885193, -1.9554768800735474, -0.6421894431114197, -0.731655478477478, -0.5033467411994934, 0.8764767050743103, 0.2986384630203247, -0.8629484176635742, -0.5704033970832825, -0.11688321083784103, 0.10687076300382614, 0.553970456123352, 0.6213012337684631, 1.8515595197677612, -0.590307354927063, -1.7274055480957031, 0.28109052777290344, -0.06749150156974792, -0.19579094648361206, -0.7850112915039062, -1.4092594385147095, 0.8539255857467651, -1.5475982427597046, -0.8673751950263977, -0.13368377089500427, -0.5333600640296936, 1.3034801483154297, 0.3603108823299408, 0.48249977827072144, -1.006432056427002, -0.02460944466292858, -0.3224646747112274, -0.4953984022140503, -0.5490997433662415, 0.7501201629638672, -0.17444685101509094, -0.37061938643455505, -1.7301431894302368, -0.18789413571357727, 1.3297576904296875, -1.0401873588562012, 0.4268922507762909, -1.097198247909546, 0.511532187461853, 0.41466236114501953, -0.44646215438842773, -1.4394338130950928, 0.5970536470413208, -1.5386042594909668, -0.5419512391090393, -0.6186660528182983, 0.5476309061050415, -0.667857825756073, -1.0949515104293823, -0.07660363614559174, -0.17190083861351013, 0.615574061870575, -0.5228646397590637, -0.6294432878494263, -1.8395074605941772, -0.8278981447219849, 1.013939619064331, -0.5545334219932556, -0.9013041853904724, -0.3304517865180969, 0.38851502537727356, 0.6762593388557434, 0.8244448304176331, 0.16595739126205444, -0.39090240001678467, 1.5134479999542236, 1.775022268295288, -1.6413847208023071, -1.119648814201355, 0.18083983659744263, -0.37536969780921936, 1.6028904914855957, 1.2132209539413452, -0.7757843136787415, 0.47696977853775024, 0.631287693977356, 0.37101590633392334, -1.4272061586380005, -0.3411843180656433, 0.15719923377037048, 0.2324392944574356, 1.8308380842208862, 0.374653160572052, 0.33692604303359985, 0.1799536645412445, 0.29196253418922424, 2.600576162338257, -1.8141028881072998, 0.866870641708374, 0.16480223834514618, 0.48909077048301697, 1.6326303482055664, -0.004586884286254644, 0.6013807654380798, -0.3763686418533325], [-0.9105314016342163, -0.20090317726135254, -2.2822725772857666, -0.6520428657531738, 2.6676247119903564, -0.3063388466835022, 0.042621299624443054, -0.4634534418582916, 1.2872107028961182, 0.01639050617814064, -0.22439107298851013, -0.5356351137161255, 0.33504074811935425, 1.8314182758331299, -1.248130202293396, -1.6963036060333252, 1.7054487466812134, -0.057548221200704575, 0.311568945646286, -0.8131557106971741, -0.8667059540748596, 1.4208165407180786, 0.8896965980529785, 0.22273565828800201, -1.000063180923462, -0.08023601770401001, 0.10054569691419601, -1.133890151977539, 0.05475988611578941, 0.38972315192222595, -0.27938443422317505, -2.1024112701416016, 0.8200177550315857, 0.6542035341262817, 1.012110710144043, -0.37852928042411804, -0.8287041783332825, 1.6252518892288208, 0.03210775554180145, -1.8204210996627808, 0.38275885581970215, 0.6539356708526611, -1.9845402240753174, -0.9216435551643372, 1.338104009628296, 0.7407345175743103, 0.21674342453479767, 1.2140445709228516, -1.3679888248443604, 0.382909893989563, -0.9918320178985596, 0.35911527276039124, 0.8240230679512024, -1.2741198539733887, -0.706084132194519, -1.6616569757461548, -0.20861893892288208, 0.5739856362342834, 2.1527884006500244, -0.1729069948196411, 0.19985182583332062, 0.15454675257205963, -0.5466386675834656, -0.7898863554000854, 0.36605316400527954, -0.6588950157165527, -0.866274356842041, -0.9922454953193665, -2.4328091144561768, 0.7612747550010681, -0.26360222697257996, -0.15819144248962402, 0.0105897793546319, -0.033449023962020874, 1.7756896018981934, 0.5241246223449707, 0.7932764887809753, 1.389523983001709, 0.907732367515564, -0.7121052742004395, 0.47485432028770447, 0.13155902922153473, -0.35495832562446594, 1.4801887273788452, 0.5130997896194458, -1.5814062356948853, 0.3831281065940857, 0.5723833441734314, 0.031051672995090485, -0.4750472903251648, -0.7904250621795654, -2.4411604404449463, -1.2543905973434448, -1.83176851272583, -0.1649453341960907, 0.7121037840843201, -0.9927001595497131, -0.24186807870864868, -0.8405930995941162, -1.2479506731033325, 0.7169595956802368, -1.254926323890686, 0.6291861534118652, -0.8010694980621338, 1.079454779624939, 0.6587393283843994, -0.20239105820655823, 0.8158583641052246, -0.43512651324272156, -0.5600654482841492, 0.6340155601501465, -1.8306291103363037, 0.9524387717247009, -0.9081577062606812, -1.864478588104248, 0.1240483745932579, -2.162412643432617, 0.23401102423667908, 0.6443668603897095, 0.9136126637458801, 2.0322225093841553, -0.06388382613658905, -2.3669838905334473, -0.9693589210510254, -0.38578158617019653, -0.9747700095176697, -1.5042078495025635, 0.3231208920478821], [-1.152817964553833, 0.7121239900588989, -0.9793249368667603, 1.8496240377426147, -0.9445121884346008, 0.30623894929885864, -1.9668132066726685, -0.19022150337696075, 1.4908150434494019, -0.07630638778209686, -1.6277387142181396, -0.5454502105712891, 0.7187318801879883, -0.5187021493911743, -0.6929386258125305, -2.0525271892547607, 0.9939894676208496, 2.2156052589416504, -0.06775761395692825, 1.5704946517944336, -0.20300406217575073, -0.864690363407135, -1.5950424671173096, 1.4425008296966553, 2.045652389526367, -0.44283729791641235, -0.18021267652511597, -1.1146634817123413, -0.34815895557403564, 0.5012924671173096, 0.1266603320837021, 1.3025380373001099, 2.15263032913208, 0.15563447773456573, -1.6588184833526611, 0.7292454242706299, 0.2123962938785553, -1.294884204864502, -1.0828800201416016, 1.626692533493042, 0.14699523150920868, 1.5081946849822998, -0.2764599025249481, 0.4439254105091095, 0.9694486856460571, -0.7257580161094666, 1.2026137113571167, 1.65218186378479, -0.2849387228488922, 1.6271504163742065, 0.5624379515647888, -1.2820067405700684, -0.5730717778205872, -0.762447714805603, -0.16986727714538574, -0.7613399624824524, -0.3198355734348297, 0.09981473535299301, 0.975123941898346, 1.9646133184432983, -0.9795780181884766, 1.3552422523498535, 2.6077747344970703, -0.16593536734580994, -1.1763999462127686, -1.5601171255111694, 1.5562852621078491, 0.05644337460398674, 1.1055082082748413, 0.38992151618003845, 0.6588464975357056, -0.33817943930625916, -1.202960729598999, -0.11097630858421326, -1.1092109680175781, -0.965462327003479, 2.0255186557769775, -1.6976429224014282, -0.2317267805337906, 0.5012519359588623, 1.142270803451538, -0.35231226682662964, 1.717118263244629, -0.3368302881717682, 0.32102686166763306, -2.186328172683716, 1.4462060928344727, -1.3482598066329956, -0.44251832365989685, -0.7713744640350342, -0.7227358818054199, -0.2885909080505371, -0.7025549411773682, 1.3094241619110107, 0.09748751670122147, 0.7398956418037415, -1.3348817825317383, 1.5125749111175537, 0.9183720946311951, 0.07487102597951889, 0.4445728063583374, 1.4842075109481812, -0.8151668906211853, -1.950803518295288, -0.7985820174217224, 0.06685107946395874, -0.5239350199699402, -1.0632070302963257, 0.09377104789018631, 0.7636808753013611, 1.5307995080947876, -0.43172261118888855, -0.02481011301279068, -0.5292566418647766, -2.242922067642212, 0.6149752736091614, -1.0286492109298706, -0.5299652814865112, 0.38857680559158325, -0.35995909571647644, -0.1886904537677765, -0.5886325836181641, 1.162929892539978, -0.2805352210998535, -1.0144529342651367, -2.10853910446167, -2.4208624362945557, 0.24210944771766663], [-0.6118433475494385, -0.6058788895606995, -0.6841298341751099, 1.7087311744689941, 0.31417664885520935, -1.9228532314300537, 1.4040738344192505, -0.9340675473213196, -0.06286679953336716, 0.8079544305801392, 0.11966830492019653, 2.014174222946167, 0.2850154936313629, -0.8826680779457092, 1.2166675329208374, -0.775115430355072, -0.30323711037635803, -1.38799250125885, 0.49774661660194397, 0.04298209398984909, 1.6493749618530273, -0.6426688432693481, 0.5394780039787292, 1.297857642173767, 0.2511897087097168, -1.6187447309494019, -2.2208943367004395, -2.132160186767578, -1.6388967037200928, 0.18228428065776825, 0.14882349967956543, 0.07071448862552643, -0.3079290986061096, -0.3212459087371826, 1.0652964115142822, -0.9209237694740295, 2.676279067993164, 0.7516860961914062, 0.9333329796791077, -1.7287665605545044, -1.7267616987228394, 0.23549455404281616, -0.3749028444290161, 0.39754927158355713, -0.40853074193000793, 0.9535879492759705, 0.23324444890022278, 0.8871498703956604, 0.10231286287307739, 0.23056219518184662, 0.7127149105072021, 0.4195939600467682, 0.769970178604126, 0.6270360946655273, -0.6205242872238159, -0.5906449556350708, -0.8434113264083862, -0.6381627917289734, -0.5078111290931702, -1.8735883235931396, -0.5435094237327576, -0.8229668140411377, -1.1229877471923828, -0.7490007877349854, -1.969103455543518, -0.3032089173793793, -0.2849541902542114, -0.37262022495269775, -0.4704088866710663, -1.598703384399414, 0.2233370840549469, -1.1272982358932495, 0.5730206966400146, 0.9709693789482117, 1.4051454067230225, 0.8882649540901184, -2.524448871612549, 2.356354236602783, 0.5453757643699646, 0.24002040922641754, 0.1222224086523056, 1.5009489059448242, -1.2982707023620605, -1.5651583671569824, 1.1655765771865845, 1.6086095571517944, 0.4529702663421631, 0.5678185224533081, 1.7679551839828491, 1.5002540349960327, -1.3104450702667236, -0.7391625046730042, 1.4914978742599487, 0.5922343730926514, 0.8567760586738586, 0.007955264300107956, -0.0727362334728241, 1.0626318454742432, -0.46118175983428955, 0.15908312797546387, -1.2590303421020508, -1.8421000242233276, -0.1888212412595749, -0.17288847267627716, 1.6810047626495361, 0.46830618381500244, 0.5937699675559998, -0.6318779587745667, 0.7232053875923157, 0.21243907511234283, 0.21978022158145905, 0.7804718613624573, -1.5223804712295532, -0.5020652413368225, -0.5589131116867065, 0.5395838022232056, -0.41445788741111755, -0.4158981442451477, -1.0340325832366943, -0.2146865874528885, 0.5128827691078186, 0.20234262943267822, -0.6963644623756409, 0.5476828813552856, 0.7170664072036743, 0.33271393179893494, -0.3923124670982361, 1.4288455247879028], [-0.2031133621931076, -1.715464472770691, -1.7810416221618652, 0.642498791217804, -0.6655020713806152, 1.799794316291809, -1.8137314319610596, 1.5011179447174072, 1.1368581056594849, 1.40720534324646, 0.20240147411823273, 1.8473138809204102, -0.0544152669608593, -0.43545040488243103, 0.9586518406867981, 0.4316682517528534, 0.6863041520118713, -0.8301643133163452, 0.5683351159095764, 0.17507661879062653, 0.16867896914482117, -0.8879278302192688, -0.07023946195840836, 0.47958648204803467, 1.59109628200531, 1.072843313217163, 0.7515246868133545, 0.3202343285083771, 0.827012300491333, -0.035874154418706894, -0.6735265254974365, 0.7202804088592529, 0.27182120084762573, 0.3784357011318207, -0.9817094802856445, -0.31520840525627136, 0.3548576831817627, -0.22309203445911407, 0.0561923086643219, 0.9432722926139832, -0.5116296410560608, 0.3720754086971283, 0.3489193618297577, -0.20175214111804962, -2.2317795753479004, 0.5166912078857422, -0.39105215668678284, -0.4218495190143585, 1.003834843635559, 0.033204879611730576, -0.24535533785820007, 1.0327610969543457, 0.7761799693107605, -0.47522038221359253, 0.1849210262298584, -0.7943980693817139, -0.5747599601745605, 1.3474812507629395, -0.7925856113433838, -0.6721049547195435, -0.37752774357795715, 0.3386014699935913, 0.9112295508384705, -0.3329279124736786, 1.5328017473220825, -0.9256547689437866, 1.0910948514938354, 0.7786988019943237, 0.05652662366628647, -0.8536621928215027, 0.6361204385757446, 1.8813241720199585, -1.3447926044464111, -0.22348976135253906, 0.7800334692001343, 0.12739849090576172, -0.9886268377304077, -2.622385263442993, -1.964718222618103, 0.29966914653778076, 0.3131829798221588, -0.7101679444313049, -0.24743607640266418, 0.8943252563476562, 1.1480494737625122, -0.08274425566196442, -0.06155785545706749, -2.0417890548706055, -1.2540878057479858, 0.684843122959137, -0.7988213300704956, 0.9919675588607788, -0.41369155049324036, -0.7353371977806091, -0.24788054823875427, 1.0665405988693237, 0.49388617277145386, -1.319715976715088, -0.05400783568620682, -0.12598799169063568, -0.39060601592063904, -0.9473543763160706, -0.5973961353302002, 0.9385213255882263, -0.8471829891204834, -0.07485691457986832, -0.32283815741539, 0.40603187680244446, -0.4270251989364624, 0.7710287570953369, 0.6739598512649536, -0.26987358927726746, 0.46363136172294617, 0.9506042003631592, 0.11000145226716995, 0.23485329747200012, -1.8164347410202026, 0.5788387656211853, -0.694950520992279, 0.27682924270629883, -1.9295233488082886, 1.1204758882522583, 0.7885861992835999, 0.4245152175426483, 1.3464351892471313, 1.1869999170303345, 0.14677174389362335, -0.6943272948265076], [0.2144579142332077, 1.7566964626312256, 1.2618604898452759, 0.676235556602478, 1.2207612991333008, 0.6458236575126648, -0.8687835335731506, 0.1195129007101059, 0.7584671974182129, 0.11715727299451828, -1.8069366216659546, -1.6108932495117188, 1.5538569688796997, 0.08571598678827286, 0.5918774604797363, -0.24150119721889496, -0.1842927187681198, -1.605973482131958, -0.7853650450706482, 1.0081287622451782, 0.5980971455574036, -1.2257843017578125, -0.8478366136550903, 1.4665522575378418, 0.2761538326740265, -1.9366453886032104, -0.9305832386016846, 0.35888731479644775, 1.208295226097107, 0.718334972858429, 1.5761975049972534, 0.29118475317955017, 1.1849831342697144, -0.6982802152633667, 0.34082773327827454, 0.10227300226688385, -0.7669007778167725, 0.02825477346777916, 0.335244357585907, 1.6798572540283203, -0.04340461269021034, -0.9736422300338745, -0.6463605165481567, 0.8037644028663635, 0.8030109405517578, 1.0194529294967651, 0.7855604887008667, 0.2534298300743103, -0.22768326103687286, 0.09479452669620514, 0.6064760088920593, -1.436638593673706, 1.2987850904464722, 1.1068195104599, -1.7022141218185425, -1.4269061088562012, 0.9147734045982361, 0.8165173530578613, -0.006360804662108421, -0.816481351852417, -1.1642062664031982, 1.035586953163147, 0.13868744671344757, 1.0477880239486694, -2.0687222480773926, 2.6256792545318604, -0.9597283005714417, 0.11124724894762039, 0.031246043741703033, 0.17409203946590424, 0.3031294345855713, -0.5757825374603271, 0.6235926747322083, -0.519220232963562, -0.9637157917022705, 0.033595163375139236, 0.47794559597969055, 0.21778111159801483, 1.166879415512085, 0.12640391290187836, 0.13032113015651703, 2.389465570449829, -0.1966424137353897, 2.095215320587158, 0.9215136170387268, 0.195261150598526, 0.2938804626464844, 0.6250391602516174, -0.8182867765426636, -1.6750627756118774, -1.1625996828079224, -0.9239198565483093, -0.3629705607891083, -1.878420352935791, 0.6331689357757568, -1.4411011934280396, -0.6230650544166565, -2.1935229301452637, -0.2679295241832733, 0.42560598254203796, -0.44841626286506653, 0.7941267490386963, -0.4508313238620758, -0.7694848775863647, -0.8954563140869141, 0.6337253451347351, 0.5673962831497192, -2.599248170852661, 1.2521783113479614, 1.3053805828094482, 1.308942198753357, 0.391265332698822, 0.5770772099494934, 1.5758765935897827, 0.9989261627197266, -1.2624807357788086, -0.5535345077514648, 0.9198969006538391, 1.8290684223175049, 0.43744099140167236, -1.474733591079712, 0.42853236198425293, -1.0298155546188354, 0.35725271701812744, 1.6537649631500244, 1.3612021207809448, -0.12109094858169556, -0.10518919676542282], [-0.08277122676372528, 0.8133459687232971, 1.0688612461090088, -1.3080512285232544, 0.7375027537345886, -1.7534087896347046, -0.3908998370170593, 0.21392197906970978, -0.15877395868301392, 0.4458068013191223, 1.6746629476547241, 1.0433225631713867, -0.7545350193977356, 0.38876160979270935, -0.3430812358856201, -0.6739863753318787, 0.905961811542511, -0.11099180579185486, -1.1468384265899658, 2.6568140983581543, -0.7805830240249634, -0.29503270983695984, 0.5342226624488831, -0.15886367857456207, -0.6639983057975769, -0.961655855178833, 0.14205926656723022, -2.3025834560394287, -0.32105204463005066, 0.5470568537712097, 1.4024357795715332, 1.0786710977554321, 0.7833623886108398, 0.2631678283214569, 1.9151726961135864, 0.32188230752944946, 0.13129176199436188, 0.3007023334503174, -0.7915018200874329, -0.7569867372512817, -0.08379948139190674, 1.1063772439956665, -0.4589964747428894, -1.3089065551757812, -0.03008231520652771, -0.06324546784162521, 0.1968315988779068, 0.9336101412773132, -0.828350841999054, 1.0580976009368896, -1.4243299961090088, 0.8417352437973022, -1.8951942920684814, 1.2324877977371216, 1.327223300933838, 1.1881650686264038, -1.4634407758712769, -0.6429325342178345, 1.2637945413589478, -1.1264756917953491, -1.8390027284622192, -0.919693648815155, 0.15909574925899506, 0.19639034569263458, -0.3692694902420044, -0.36247992515563965, 0.14522883296012878, 0.5151216387748718, 1.1861073970794678, 0.6849689483642578, 1.7961698770523071, -0.6267195343971252, 0.563563883304596, -1.248917579650879, 2.130310535430908, -0.2840087115764618, 0.39622244238853455, -0.44139790534973145, -0.6196508407592773, -0.9548676609992981, 0.24096433818340302, -1.8095424175262451, -0.6183148622512817, -0.31891411542892456, 2.171353816986084, 1.2713770866394043, -0.559280276298523, 0.5079584121704102, -1.104630947113037, 0.2989647686481476, 2.2510080337524414, -0.5689382553100586, 0.7511292695999146, 0.6094484925270081, 2.596264600753784, -0.06863006949424744, -0.8208175897598267, -0.8395764231681824, 1.3257173299789429, 1.4450325965881348, -0.7052645087242126, 0.8490496277809143, -1.6128580570220947, -1.3334100246429443, -1.4445931911468506, 1.8988436460494995, -0.7597050666809082, 0.13870616257190704, -0.6566717028617859, -0.7999795079231262, 0.9161999225616455, -0.9268160462379456, 1.2868238687515259, 0.1503177136182785, 0.19482247531414032, 0.7180473208427429, -0.06170964241027832, -0.7655279636383057, 0.12776897847652435, 0.5463941693305969, -1.5473484992980957, -1.7756884098052979, 0.606243908405304, 1.0311498641967773, 0.11542879045009613, -0.49856576323509216, -0.6197601556777954, 2.45975923538208], [0.16852125525474548, 0.21376873552799225, 0.7685385346412659, 0.10756267607212067, -0.1447025090456009, 1.6444956064224243, -1.8667218685150146, -0.8854793310165405, -0.041395414620637894, -1.7799229621887207, -0.3977336883544922, -0.8134528398513794, -0.5506648421287537, 0.17382338643074036, 0.844910204410553, -0.5348172783851624, -1.059718132019043, -0.9175019264221191, 0.8494371771812439, 2.1126644611358643, 0.9810083508491516, -0.18326088786125183, 0.49365201592445374, -0.11105088889598846, 0.7441081404685974, -1.9242193698883057, 0.45008957386016846, -0.44771069288253784, -0.12016132473945618, -0.5127711296081543, 1.8316223621368408, -0.13115109503269196, -0.6837080717086792, -0.8857430219650269, 1.6212860345840454, -0.23301193118095398, 0.5368970036506653, -0.03460971266031265, -0.09355808794498444, 0.3434242308139801, 0.6631225943565369, -1.5239486694335938, 0.8066328167915344, 0.45345553755760193, 1.3501334190368652, -1.3620796203613281, -2.74519419670105, -0.04061080142855644, 0.3411915898323059, 0.5360943675041199, 0.6489567756652832, 0.7328045964241028, 0.5471231937408447, 1.2221813201904297, 1.2802304029464722, -1.6699895858764648, 0.6530599594116211, -1.267992377281189, -0.6445614695549011, -2.028425455093384, -2.089177131652832, 0.008868901059031487, 0.4485919773578644, 0.547404408454895, 0.5364577770233154, 0.1707131266593933, -2.004603624343872, -0.16495303809642792, 0.17780454456806183, -1.4642950296401978, 1.5376945734024048, 0.15774306654930115, -1.4286150932312012, 0.04388671740889549, 0.1565510481595993, -2.0175461769104004, -0.8746540546417236, 1.2814490795135498, 0.5306035280227661, -0.5200990438461304, 0.8168221712112427, 1.976423740386963, -0.9973337650299072, 0.18354563415050507, 1.6083945035934448, -1.4584790468215942, -0.8598293662071228, 0.3721175789833069, 0.3593882918357849, -0.4966312050819397, -0.0624908022582531, 0.646994411945343, 0.3656301498413086, -0.5618025660514832, 0.7157450318336487, -0.20908398926258087, -1.6148940324783325, 0.281546950340271, -0.34029537439346313, 1.2538481950759888, -1.4358052015304565, -0.7568303942680359, -0.5220609307289124, -0.46480593085289, 0.5629020929336548, -1.3677754402160645, 0.8821718692779541, -0.10191567242145538, -0.07955906540155411, 0.10538320243358612, 1.3198927640914917, 0.1404067575931549, -2.0347392559051514, -0.12105637788772583, 1.1316603422164917, -0.7432109117507935, -0.6358025074005127, -0.028526704758405685, 1.1316862106323242, -0.6483506560325623, -0.8540380001068115, -0.30415043234825134, -0.9979361295700073, -0.32551485300064087, -0.9141184687614441, 0.8229894638061523, -1.5098788738250732, 0.52064448595047], [-0.3332837224006653, 0.48568591475486755, -0.8513002991676331, 0.025790279731154442, -0.25282159447669983, -0.6498600244522095, 0.7647286057472229, 0.047943104058504105, -0.26786699891090393, -0.2409803569316864, -0.41367846727371216, 0.022154171019792557, -1.0229699611663818, -1.7754849195480347, 0.5907567143440247, 1.1770176887512207, 0.5244513750076294, -0.6863221526145935, -2.0507164001464844, 1.188969612121582, 0.3417414128780365, -0.21024538576602936, 1.6958988904953003, 1.4340534210205078, -0.4728301763534546, -0.015087508596479893, 0.7326274514198303, 0.18599605560302734, 0.004216082394123077, -0.15842105448246002, 0.46664226055145264, -0.6191591024398804, -0.11764752119779587, 1.992884874343872, 1.2098397016525269, 0.975818395614624, 0.41499727964401245, 2.015617847442627, -0.5281720757484436, 0.6532913446426392, 0.04487502574920654, 0.7429768443107605, -0.0791938379406929, 0.4924663305282593, -0.6162667870521545, -1.7698335647583008, -0.23364289104938507, -0.5281173586845398, -0.3781423270702362, 1.8807092905044556, -0.2251707911491394, 1.44120192527771, -0.08349794894456863, -0.0649060532450676, -0.09866354614496231, -0.37215930223464966, -0.1731521338224411, 0.2677215337753296, 0.9855920076370239, -0.7536647319793701, -0.5540730953216553, 1.822685718536377, -1.324308156967163, -0.6706125140190125, 0.9528647065162659, -0.2424391210079193, -0.004187162034213543, -0.4343357980251312, 2.136535406112671, 0.8474546074867249, -0.79111248254776, 1.1547719240188599, -0.657139241695404, 2.3629379272460938, 0.23087343573570251, 1.3661073446273804, 0.7392513155937195, -0.8509790301322937, -0.30765774846076965, 0.4174558222293854, 0.5846295356750488, -0.06597980111837387, 1.0139487981796265, -0.1136636883020401, 0.937192440032959, 0.03476899862289429, -1.8579734563827515, -1.4884389638900757, 0.91218101978302, 1.286301851272583, 0.03243931010365486, 0.05168618634343147, -1.1759830713272095, -2.4790053367614746, 0.37052467465400696, -0.8378869295120239, -1.0915191173553467, 0.8092694878578186, 0.8044745326042175, 0.6159034371376038, 0.7888602614402771, -2.0624523162841797, 1.4387731552124023, -0.5507093071937561, -0.1174127385020256, -0.02279568463563919, 0.45677435398101807, 0.9372723698616028, 0.10606850683689117, 0.425182580947876, 0.016939232125878334, -0.467816561460495, 2.019380807876587, 0.8467051982879639, -0.30028170347213745, -1.9003257751464844, 0.8736062049865723, 0.07135073095560074, -2.064303159713745, -0.2851700186729431, 0.18110370635986328, -1.8854328393936157, 0.8927974700927734, -0.6589922904968262, -0.12001770734786987, 0.7429921627044678, 0.5561531782150269, -0.4863226115703583], [-0.9617159366607666, 0.7606686949729919, -1.0941797494888306, -1.1447339057922363, -0.4180765450000763, -1.0276299715042114, 0.12527881562709808, -0.6667481064796448, -0.03613422065973282, -0.0017478590598329902, 0.6796318888664246, -0.6771955490112305, 0.15512946248054504, 1.6062204837799072, -0.020456762984395027, 0.2479541152715683, -0.30669960379600525, -2.445474147796631, 0.1339263767004013, -1.2531933784484863, -2.5371406078338623, 0.531068742275238, 0.3360680639743805, 0.01685747504234314, 1.114173173904419, -0.6109368801116943, 0.4901731014251709, -0.0629151463508606, -0.3048573136329651, 0.025982074439525604, 3.3536665439605713, -0.879210889339447, -0.8131077885627747, 1.9834791421890259, -0.45222601294517517, 0.9053558707237244, -1.1322262287139893, -0.8295072913169861, -0.05165999382734299, -0.9601263999938965, 1.7235177755355835, 0.9881417155265808, -0.7252269387245178, 0.8329970240592957, 1.9321664571762085, -0.8417670726776123, 0.9400091767311096, 1.6269574165344238, 0.6903408765792847, -1.023458480834961, 1.479557991027832, -0.9038741588592529, -0.38721734285354614, -0.08973219990730286, 0.21594545245170593, -0.0720149353146553, -1.035187005996704, 0.32388997077941895, 1.1718392372131348, 1.0255929231643677, -0.36787354946136475, 0.2069329172372818, -0.5991430282592773, -1.5521001815795898, 0.39805543422698975, 0.732857346534729, -0.8724908232688904, 1.1938287019729614, -1.745889663696289, 1.0613893270492554, -0.04551248624920845, 0.5817890763282776, 0.802965521812439, -0.2163657397031784, 1.4172590970993042, 0.2088811844587326, 0.32998719811439514, 1.3657630681991577, 0.3118728995323181, -0.2162286341190338, 0.258657306432724, -0.7468129396438599, 1.070051908493042, 0.14294573664665222, -0.007633361965417862, -2.197347402572632, -0.6096716523170471, 0.03490158170461655, -1.835982084274292, -0.7124918103218079, -0.6218622326850891, 0.15312404930591583, 1.2140737771987915, 1.4172875881195068, 1.07174551486969, 0.05287383869290352, -1.8740464448928833, -1.61687171459198, -2.1077945232391357, 0.7121168375015259, 0.15584249794483185, 1.8638815879821777, 0.6229967474937439, 0.5770586729049683, -0.7255716919898987, 0.020974785089492798, -0.1782006472349167, -0.8429909944534302, -1.156489372253418, 0.5221620202064514, 0.11978434026241302, 0.22623394429683685, 1.594601035118103, -0.36482298374176025, -2.2804574966430664, -0.8655847907066345, -0.011893420480191708, 0.8628532886505127, 0.2973635494709015, -1.0526460409164429, 0.18307609856128693, 0.5691267848014832, -0.14455826580524445, 1.7691397666931152, 1.067015528678894, -0.5350192189216614, -0.6110090613365173, -0.4524799883365631], [0.17737628519535065, 0.9398983120918274, 0.9550737738609314, 0.28672122955322266, 0.26303941011428833, 0.8842097520828247, -1.0989311933517456, 0.6848292350769043, 1.3906340599060059, -0.4261351525783539, 0.6027165651321411, 0.024644285440444946, 1.3929935693740845, -1.8233762979507446, -1.076629638671875, 1.6710319519042969, 0.9370104074478149, -0.567313015460968, -0.6442562937736511, -1.285386085510254, -1.5443267822265625, 0.8810034990310669, 0.9014950394630432, 1.088967204093933, -0.29758307337760925, -0.5773048400878906, 0.1623765081167221, 0.13413889706134796, -0.4530049264431, 1.1780986785888672, -0.2636674642562866, -0.4609450697898865, -0.38190528750419617, 0.039555877447128296, 0.08096053451299667, 1.0054510831832886, 0.5170037746429443, -0.9818986058235168, -0.569500207901001, 0.03730716183781624, -0.02930581569671631, -1.9425996541976929, 1.313007116317749, 0.3827784061431885, 0.6921605467796326, -1.689347267150879, 0.7383790016174316, -1.1609666347503662, -0.09553051739931107, 0.22987565398216248, -0.1715513914823532, -1.7808054685592651, -0.3319711983203888, 1.3640447854995728, -0.4070572853088379, 0.040744900703430176, 0.9818543791770935, 1.0336380004882812, 0.8153663277626038, -0.22455981373786926, -0.8378245234489441, -0.36107781529426575, -0.2022595852613449, -0.38413649797439575, 0.29057958722114563, 0.8171990513801575, 1.0515446662902832, -1.7453464269638062, -0.13898271322250366, 1.910731554031372, 0.5131274461746216, -0.10094904154539108, -0.21557524800300598, -1.352126121520996, 2.4440536499023438, 0.6709514260292053, 0.29932689666748047, 1.1259597539901733, 0.22806091606616974, 0.2377564013004303, 1.4675253629684448, -0.9856046438217163, 0.5175808668136597, 0.44129812717437744, 0.803045392036438, -0.35349205136299133, 0.2781578302383423, -0.8899497985839844, 1.49209725856781, -0.5112090706825256, -1.1799424886703491, -0.1869848370552063, 0.24369387328624725, -2.070181131362915, -1.1111005544662476, 1.8412666320800781, 0.5305989980697632, 0.7425016164779663, -1.5798193216323853, 0.04180947318673134, -2.8761794567108154, -1.7137681245803833, 0.5442008376121521, -0.3374609351158142, 0.03295600041747093, 0.7186251282691956, -0.36354124546051025, -0.6696355938911438, 0.9144584536552429, 0.0460190586745739, 0.38676586747169495, -0.9054570198059082, 1.0510927438735962, 1.3222764730453491, -0.6273482441902161, 0.37920069694519043, -0.7384403944015503, -2.3682241439819336, -0.5575131177902222, 1.244197964668274, -1.1132920980453491, 0.47926831245422363, 0.40555837750434875, -0.4622422158718109, -0.5964568257331848, -0.4554932117462158, -0.43889856338500977, 0.047143541276454926], [1.4808895587921143, 0.8420657515525818, -1.626239538192749, -0.10517050325870514, -0.9558817148208618, 2.989985227584839, -1.2079617977142334, -1.0792185068130493, -0.43539753556251526, -0.1858816146850586, 0.28330111503601074, -1.2601919174194336, 1.112929344177246, -1.54213547706604, 0.7794209122657776, 0.8847686648368835, -1.3056697845458984, -0.046440694481134415, 0.04182368516921997, 1.3638930320739746, -1.7784149646759033, 0.5098473429679871, 1.1905103921890259, -0.008237181231379509, -0.7178075909614563, -1.1871795654296875, -0.08180559426546097, 0.9476657509803772, -0.28260162472724915, 1.2563862800598145, 1.1434251070022583, 0.8800227642059326, -0.1685526967048645, -0.5058008432388306, -0.5094457864761353, -1.1790812015533447, 0.9217672348022461, -1.8131129741668701, 0.8342915773391724, 1.0004085302352905, 0.7733122706413269, -0.16861525177955627, -0.6514546275138855, -0.5834864377975464, -0.40949201583862305, -1.1729384660720825, -1.4014216661453247, -0.03928855434060097, -0.5407134890556335, -0.7358757257461548, -1.2911173105239868, 0.6190162301063538, 1.0717543363571167, -0.08577986806631088, -0.18492655456066132, 0.27672404050827026, -1.7621132135391235, 0.6776621341705322, 1.38975191116333, -0.7594794631004333, -0.4551265835762024, -2.994986057281494, 0.8116603493690491, 1.7391836643218994, -0.46738573908805847, -1.1334677934646606, 0.22729118168354034, 0.47580352425575256, 0.02019021287560463, 0.7893097996711731, -0.4604949951171875, -0.6544913649559021, 0.9536483883857727, -0.9873945116996765, 1.2232671976089478, -0.31159505248069763, -0.28857681155204773, -0.4163311719894409, -0.1898910403251648, 1.0505778789520264, -2.045649528503418, -0.24809013307094574, 1.170352578163147, -1.3683544397354126, -1.0042052268981934, -0.41697677969932556, -0.3843361437320709, 0.9104015231132507, -1.334091305732727, -0.26921242475509644, 0.2136458158493042, -0.4064469635486603, 0.22652588784694672, 0.6511128544807434, -0.21183830499649048, 1.3719353675842285, -0.6811701655387878, 0.795437753200531, 0.31017664074897766, -0.47368526458740234, 0.29703426361083984, 0.6192795038223267, -0.8804083466529846, 0.8147116899490356, -2.1425061225891113, 0.21162405610084534, -0.0556454062461853, -0.7631922960281372, -1.6893432140350342, -0.798453152179718, 1.4496262073516846, -1.1085350513458252, -1.3045940399169922, 0.3645328879356384, 0.0825486034154892, 0.3228631615638733, -2.117833137512207, -1.6097203493118286, 0.3630817234516144, -0.9106222987174988, 0.5579048991203308, 0.3936103880405426, -0.710869312286377, 1.4331682920455933, -0.20488253235816956, 1.5255228281021118, -0.6051471829414368, -0.16051432490348816], [-1.4403071403503418, 0.30475491285324097, -0.25446850061416626, 0.043135277926921844, -1.1260489225387573, 0.9291077256202698, 0.5701921582221985, 2.1568360328674316, -1.2200524806976318, 1.069089412689209, 0.6228311657905579, 1.7694939374923706, -0.113040491938591, 1.7184631824493408, -1.2000497579574585, -0.5417916178703308, -0.010445977561175823, -0.05290314927697182, 1.1762547492980957, 0.05683944746851921, 0.0896603986620903, -0.5674829483032227, 1.5785119533538818, 0.46948790550231934, 1.130268931388855, 1.4809621572494507, -0.3103008270263672, -1.3702846765518188, 0.6222398281097412, -0.650010347366333, 0.6722918152809143, 0.17809663712978363, -0.5683926343917847, 0.28909921646118164, 0.2470308542251587, 1.203947901725769, -0.359724760055542, 0.12837612628936768, 0.3649502098560333, -2.1680233478546143, 0.9616222381591797, 2.051955223083496, -0.7038936614990234, 0.2833254933357239, -0.5273694396018982, 1.2254385948181152, -0.7282680869102478, -1.2966477870941162, 0.37879887223243713, -0.13047996163368225, 1.4512269496917725, 0.0072868335992097855, -0.6347216367721558, -0.637031614780426, 0.5597071051597595, -0.22433485090732574, -1.5696200132369995, -0.8534618616104126, 0.3220442533493042, -0.8922898173332214, -1.227384090423584, 0.34257519245147705, -0.512272834777832, 0.0392967090010643, -0.4449959397315979, -0.660944402217865, 0.1712092161178589, -0.9358526468276978, 3.1090002059936523, -1.216978907585144, 0.24106639623641968, 0.09196121990680695, -1.8867342472076416, 1.7459319829940796, 0.14930924773216248, -0.06354479491710663, 1.1158770322799683, 0.8289843797683716, 0.3031555712223053, -0.6399742364883423, -1.3552182912826538, 1.063104271888733, -0.3660580515861511, -0.4261844754219055, 1.179606556892395, -0.04156917333602905, 0.9127083420753479, 1.2409306764602661, -0.520472526550293, -0.49983885884284973, -0.04653138294816017, -1.6271893978118896, -0.2603488564491272, -0.40934646129608154, 0.5895119905471802, -1.6243529319763184, 2.836608648300171, 0.16952796280384064, -0.17495498061180115, -0.5971518158912659, -1.6194149255752563, 0.01353242713958025, 0.21649383008480072, 1.3770325183868408, 0.9535561800003052, -0.871005117893219, -0.3055099844932556, 0.6029025316238403, 1.2397809028625488, 1.6807864904403687, -0.7978819012641907, -0.44742077589035034, 1.2737407684326172, 1.1670955419540405, -0.27547577023506165, -0.27563557028770447, -0.56696617603302, 0.17259496450424194, 0.9615843296051025, 0.9417611360549927, -0.3054574429988861, 0.06830188632011414, -1.6851871013641357, -1.6924123764038086, -1.3606948852539062, 0.8240677118301392, 0.6721730828285217, -1.5017650127410889]]], [[[0.694597601890564, 1.6241843700408936, 0.271711528301239, 0.8427097201347351, 2.6756579875946045, -1.9542969465255737, 0.38914889097213745, -0.20871879160404205, -1.390931487083435, 0.8214238882064819, 0.4071633219718933, 0.47290417551994324, 0.6904619336128235, 0.9990369081497192, -0.14376047253608704, -0.2217620313167572, 1.3266035318374634, 2.4469234943389893, -0.31675949692726135, -0.1094452366232872, 1.4317902326583862, -1.5216563940048218, 0.2930331528186798, 1.2297708988189697, 1.0723649263381958, -0.21114031970500946, -0.7982872724533081, 0.7888241410255432, -1.2142044305801392, 0.34392574429512024, -1.2331520318984985, -0.123710997402668, 0.2390526980161667, -0.6554268598556519, -0.1073836162686348, -0.9479185342788696, -0.5875685811042786, -0.09348037838935852, -0.00675663398578763, 1.9564380645751953, 0.3272824287414551, -0.08159850537776947, 0.6819081902503967, 0.7540727853775024, -1.3884588479995728, -0.18423695862293243, -0.3524768352508545, 0.5867161154747009, -0.8385539650917053, -1.4485805034637451, 1.1110316514968872, 0.3494747579097748, 1.332360029220581, 1.3255491256713867, -0.5148575901985168, -0.0443541519343853, -2.6855976581573486, -1.6164923906326294, -0.3101355731487274, 0.8099266290664673, -0.29828551411628723, -1.2894301414489746, -0.45951950550079346, 0.8941766023635864, 0.6796748042106628, -2.0100808143615723, 0.37919721007347107, 0.43443092703819275, -0.7802673578262329, 0.4666885733604431, -1.0782032012939453, -0.3319030702114105, -2.124166250228882, 0.2705543339252472, 0.6211799383163452, 0.9241507649421692, 1.863283395767212, -2.1955926418304443, 0.019335005432367325, 0.020697053521871567, 0.7772040963172913, -1.6268959045410156, 0.7732356190681458, -0.04902360588312149, -1.3600696325302124, -0.18251453340053558, -0.2815423011779785, 0.4019791781902313, -0.5832284688949585, -1.0539376735687256, 0.1622142642736435, -1.4327815771102905, -0.36811327934265137, 1.2753114700317383, -2.386746883392334, -0.7086278200149536, -0.8557430505752563, 1.4815666675567627, 0.25732576847076416, 0.8529709577560425, -0.6580915451049805, 0.03188977390527725, 1.7419861555099487, 1.320378303527832, -1.2601590156555176, -1.1412785053253174, 0.2817009687423706, 0.4772505462169647, 1.5121445655822754, -1.295815348625183, -0.29891499876976013, 0.8306915760040283, -1.31504487991333, 1.8706961870193481, -0.5617458820343018, 0.6457124352455139, -0.021569564938545227, -0.13501453399658203, -1.1485093832015991, 0.8995501399040222, -0.9102432131767273, -0.24036730825901031, 0.697221577167511, 0.5184691548347473, -0.2739841043949127, -1.6952487230300903, 0.033343516290187836, 1.5663540363311768], [0.365113765001297, 2.7541375160217285, 1.1650841236114502, -0.5700231790542603, 1.047511100769043, -0.14610803127288818, 0.08176425844430923, 0.5618574023246765, -0.6810012459754944, 1.392762303352356, -0.5005173087120056, 0.6470136046409607, 1.5359094142913818, 0.04591406509280205, 0.5710014700889587, 0.2204350382089615, 0.9594249725341797, -0.5094887018203735, 0.383994460105896, 0.11522045731544495, -0.7885671854019165, -0.43682125210762024, 0.5603554844856262, 0.07671641558408737, 1.599774718284607, 0.07854816317558289, -1.1026737689971924, -2.1710357666015625, 0.21405884623527527, -0.5471674203872681, 1.1074074506759644, 2.0255770683288574, -0.736802339553833, -0.34846198558807373, 0.5439313054084778, -1.2315257787704468, -1.101053237915039, 0.6101252436637878, -0.7055027484893799, 0.06506949663162231, 0.5007779598236084, 1.112064003944397, -0.6224310398101807, -0.9899448156356812, 0.28705549240112305, 0.04574573040008545, 0.6523086428642273, 0.9162456393241882, -0.2928774952888489, 0.5844024419784546, 0.263195276260376, 0.2338249832391739, 0.6354791522026062, 1.300286054611206, 1.467141032218933, 1.2126678228378296, 0.1713566780090332, -0.08755464106798172, -0.47568926215171814, -1.171004295349121, 0.1865834891796112, -0.4971311688423157, -1.9044467210769653, -0.8355363607406616, -1.402133822441101, -1.306223750114441, -1.2182706594467163, 1.3512215614318848, -0.1704508662223816, -0.2849326431751251, 0.15542493760585785, 0.9270729422569275, -0.8466202020645142, -1.4296529293060303, -0.8787830471992493, 0.09170734137296677, 0.5675197839736938, -1.022139549255371, 0.5729379653930664, -1.2522484064102173, 0.8990680575370789, 1.6411616802215576, -0.8114784359931946, 0.577465295791626, 1.4545142650604248, 0.6723246574401855, 1.064023733139038, -1.2059568166732788, -1.00542414188385, -1.6704350709915161, 1.3274660110473633, 0.3783557415008545, -1.4180787801742554, 0.05229923874139786, -0.05259767174720764, -2.2601068019866943, -0.36448222398757935, -0.25092801451683044, 0.6357272267341614, -1.4711405038833618, 0.8286300301551819, -0.45943930745124817, -1.516377329826355, -0.33007705211639404, 0.09146931022405624, 0.013345059007406235, -0.04592134431004524, -0.9080505967140198, 0.7951149344444275, -0.39231038093566895, -1.0861608982086182, 1.4005829095840454, -0.6998178362846375, 0.5228323340415955, -0.09623589366674423, 0.9927820563316345, 0.7148724794387817, 0.2520262897014618, 0.9897408485412598, -0.5987343192100525, 0.650937557220459, 0.4527132511138916, -0.6523313522338867, 0.1440502107143402, 0.6414507031440735, 0.7546187043190002, 0.8095828294754028, -0.18528752028942108], [-0.6885858774185181, 0.1576734334230423, -0.3492845594882965, -0.648830771446228, -0.8629646301269531, 0.34245991706848145, -0.2058829814195633, -1.981561303138733, -1.8036218881607056, -1.2530232667922974, 1.4254162311553955, -0.6172107458114624, -2.062326669692993, 0.9393829703330994, -0.3318127691745758, 1.1789113283157349, -0.37260007858276367, -0.7914496064186096, 1.0712591409683228, 1.0945429801940918, -0.6546511054039001, -0.8679757714271545, -0.32317474484443665, -0.49724987149238586, -1.5865927934646606, 0.5607123374938965, -0.16403955221176147, -1.053196668624878, -0.7564137578010559, -0.9378822445869446, -1.7148340940475464, -0.06928329169750214, 1.512778401374817, -1.1813396215438843, -0.32118725776672363, 0.520592987537384, 1.1418036222457886, -1.6788922548294067, -0.2474033236503601, 0.6354559063911438, 1.5387098789215088, -1.2897669076919556, 0.058183956891298294, 0.7506427764892578, -0.4436027407646179, -1.5621920824050903, -1.1267703771591187, -0.39598825573921204, -2.7786812782287598, -0.18605628609657288, -0.7124753594398499, -0.794702410697937, -0.061875373125076294, -1.4126102924346924, -1.3962368965148926, 0.6075954437255859, -1.2000092267990112, 0.9577990770339966, -1.6809055805206299, 0.9517614841461182, -0.5516961812973022, -2.6227948665618896, -0.24358417093753815, -0.7847016453742981, -1.0602258443832397, -1.8657475709915161, 2.258964776992798, -0.9862587451934814, 0.670644223690033, -0.4163394272327423, -0.18751533329486847, -0.42384079098701477, 0.5526183843612671, 0.7613446712493896, 0.09999853372573853, 0.45544981956481934, -1.1457542181015015, -1.1815729141235352, 0.5391883254051208, 0.26988986134529114, 0.038884665817022324, -1.555314540863037, 0.13799796998500824, -1.003246545791626, -1.5212889909744263, 0.0493612177670002, 0.4616903066635132, 0.9995251297950745, -0.1237037181854248, -1.3682416677474976, -1.381839394569397, -1.1636441946029663, 1.9402620792388916, -0.2120586484670639, 0.6012899279594421, -0.7928561568260193, 0.4643881916999817, 0.5445193648338318, 1.4147388935089111, -2.2697255611419678, -0.7133532166481018, 0.8994521498680115, 1.5975289344787598, 0.03792396932840347, -0.989917516708374, -0.4297875761985779, 0.4674808382987976, 1.8871002197265625, 2.4781880378723145, -0.5480957627296448, 0.027899151667952538, 0.5014910697937012, -0.5462809205055237, 0.21356341242790222, 0.3249742090702057, 0.3418107032775879, -0.5826072096824646, -0.1653284728527069, 0.786207914352417, 0.09208457916975021, 0.9025844931602478, -0.4158799350261688, 1.3001445531845093, -1.6411033868789673, -0.4281521439552307, 0.6538912057876587, -0.48124754428863525, 0.05564876273274422], [-0.7406721115112305, 1.582423210144043, 0.2804749011993408, 1.7022242546081543, 0.008085341192781925, -1.206142544746399, -0.0050483266822993755, 0.48310279846191406, -0.34217172861099243, -1.3263804912567139, -0.3682749271392822, 0.45643481612205505, 0.06572233140468597, 0.522909939289093, -0.47744420170783997, -0.492318332195282, -0.2926599383354187, -0.9706887602806091, 0.6822736859321594, -0.1862850934267044, -0.4935319423675537, -1.8696041107177734, 1.520656704902649, 0.7602757215499878, 1.3656346797943115, 0.6660957932472229, 0.2731966972351074, 1.348845362663269, 1.1619524955749512, 0.8923229575157166, -0.055618226528167725, 1.1464951038360596, -0.8234602212905884, -0.21579086780548096, -0.20846469700336456, 1.5373905897140503, -1.4347736835479736, 0.48211535811424255, 0.887424647808075, -0.5107522010803223, 1.1650092601776123, 1.0510224103927612, -0.5957515239715576, 0.5023314356803894, 0.21414940059185028, -0.5456626415252686, 0.9205114245414734, 1.5783919095993042, 0.14335478842258453, 0.5125710368156433, 0.7305643558502197, 0.6293402910232544, 0.7231713533401489, 0.026570651680231094, -1.0316753387451172, -1.4776426553726196, 0.17107385396957397, 1.070908784866333, 0.5028976202011108, -1.6438788175582886, 1.0810514688491821, 1.9484344720840454, -0.4604937434196472, 1.91244375705719, -0.625219464302063, 0.9828841090202332, 0.46205341815948486, -0.577941358089447, -0.07492540031671524, -1.376874566078186, -0.6850016117095947, -0.4628077745437622, 1.3581924438476562, 0.6985734701156616, -0.03891986981034279, -0.9059539437294006, -0.9798856973648071, -0.6982322931289673, -0.469614714384079, -1.191239595413208, -0.49114155769348145, -0.6805706024169922, 0.44959190487861633, -0.10374956578016281, 1.76322603225708, 0.6614618301391602, -0.8190785050392151, 0.08409833163022995, 0.18005596101284027, 0.09273548424243927, 0.07609345018863678, 0.32160645723342896, -1.5591464042663574, 0.4963379502296448, 1.2396597862243652, 0.9960579872131348, 0.24849580228328705, -1.1654530763626099, -0.5644410848617554, 1.7834221124649048, 0.19525539875030518, 0.11628792434930801, 1.3994449377059937, -0.6255581974983215, -1.2637531757354736, -0.40431728959083557, 0.07681620121002197, -0.606998085975647, -0.061173953115940094, -0.7063092589378357, -1.0016547441482544, -1.486771821975708, -0.19257864356040955, -0.9642943739891052, 0.5192550420761108, -0.9113020896911621, -0.25862377882003784, -0.0956132635474205, 1.3725451231002808, 0.9550104141235352, -1.4508213996887207, -0.13034410774707794, -0.073174849152565, 0.09624586999416351, 2.5719244480133057, -0.5039286017417908, 0.41451480984687805, 0.9662919044494629], [1.6367675065994263, 0.1930573284626007, -0.4841128885746002, 1.0324641466140747, -0.7762526869773865, 0.026543162763118744, -0.7995434403419495, -1.0166560411453247, 0.8487817049026489, -0.5653064250946045, -0.5182203054428101, 0.9276032447814941, -0.5782363414764404, -1.1211140155792236, 0.7180634140968323, 0.1506175845861435, 0.6095345616340637, -0.4053598642349243, 0.28209391236305237, -2.2448034286499023, -0.6512717008590698, 0.21628481149673462, 0.4095534086227417, -1.4522173404693604, -0.5826204419136047, -0.9116458296775818, -1.9864377975463867, -1.4331660270690918, 0.7656973600387573, -0.7985132336616516, -0.5518727898597717, -0.5622420907020569, 0.588732898235321, -0.06006360054016113, 0.3432849049568176, 3.1819918155670166, -0.5707706809043884, 0.6994975805282593, -0.7224102020263672, 2.8103506565093994, -1.4793320894241333, 1.0449721813201904, 1.3125815391540527, 0.6765866279602051, -0.40817901492118835, -1.5759400129318237, -0.7438948154449463, -0.8403927087783813, -0.15396563708782196, -0.20150746405124664, 0.37323060631752014, 0.2682294249534607, 0.1163637563586235, 0.49951741099357605, 0.3077983260154724, -0.6923303008079529, -1.592651128768921, -0.3766837418079376, 0.4993945360183716, -0.34841182827949524, -0.7142385244369507, -1.2000606060028076, 0.7438936829566956, 1.1623802185058594, -0.8381606340408325, 0.3173104226589203, 0.37081456184387207, 1.4417665004730225, -0.28424209356307983, 1.9748817682266235, 0.9103437066078186, 0.01725364476442337, 0.9083963632583618, -0.0727793499827385, 0.9816542267799377, 0.6557446718215942, 2.0715584754943848, -1.0441240072250366, -0.40638288855552673, -0.6087979078292847, 0.10371941328048706, 1.3024122714996338, -0.21752344071865082, -0.3999018371105194, -0.30306291580200195, -2.50618839263916, -1.1747075319290161, -1.9015687704086304, 1.0872976779937744, 1.112949252128601, -0.21506060659885406, -0.7269430756568909, 0.09226596355438232, 0.07038800418376923, -0.5131371021270752, 0.26350346207618713, -0.8993350267410278, -0.14868615567684174, 0.2447170466184616, 0.4512706696987152, -0.3415824770927429, 0.5279960036277771, -2.9506309032440186, 0.8308630585670471, -0.2946239113807678, 0.8625805377960205, 1.2404439449310303, 0.664039134979248, 0.5300482511520386, 0.6634427905082703, -0.8159856796264648, 0.2332080602645874, -0.7883674502372742, 0.19079457223415375, 0.542446494102478, -0.19139279425144196, 0.5738614797592163, -0.34188321232795715, 1.4984830617904663, 1.1045833826065063, 0.36326542496681213, 0.4148181080818176, 0.39044487476348877, -1.1938457489013672, -1.6278202533721924, -0.2489711195230484, 0.19698266685009003, 0.11883818358182907], [-0.5643954873085022, -0.40714317560195923, -0.3789261281490326, 0.5588542819023132, 1.097909688949585, -0.7456371188163757, 0.9190680384635925, -0.2572667896747589, -0.042310670018196106, -1.3589199781417847, -1.6867941617965698, 0.5221571326255798, -0.805461049079895, -0.5801713466644287, -0.2623284161090851, 1.3173037767410278, -0.36583301424980164, -1.4694766998291016, -1.7434067726135254, -0.43297141790390015, 0.9368719458580017, 0.008283091709017754, -1.4188897609710693, -0.6721267700195312, 2.265780210494995, 0.04707152023911476, -0.4313720464706421, 0.048797592520713806, -2.537583112716675, -0.42223313450813293, -0.35793009400367737, -0.651973307132721, -0.8502048850059509, 1.06147301197052, 0.0789799690246582, 0.2748014032840729, 1.293631672859192, 0.3608343005180359, -0.12206748872995377, 0.06763023138046265, 0.8593882322311401, -1.0771211385726929, 1.2378859519958496, 0.15697261691093445, -1.6425431966781616, 1.3085705041885376, 0.33368465304374695, 1.2039158344268799, -0.2708342373371124, -0.26634570956230164, -0.7738552689552307, -0.05197455361485481, 1.7851130962371826, -0.42814740538597107, 1.2863216400146484, 0.23259569704532623, 0.04707977548241615, -0.5206015706062317, -0.5264648199081421, -0.7933592200279236, -0.09087410569190979, 0.042632024735212326, 0.9676347374916077, -1.3107272386550903, -0.792434573173523, -0.04817965254187584, 0.753300666809082, -2.1560232639312744, 1.0040727853775024, 0.6263010501861572, -0.8202541470527649, -1.5075576305389404, -0.9547789692878723, 0.9703930616378784, 0.3485026955604553, 1.2877625226974487, -0.03205733746290207, 0.08921747654676437, 0.6140514612197876, -0.8580000996589661, -1.233186960220337, 0.8491997122764587, -0.22802254557609558, 0.93919438123703, -1.1782068014144897, 0.31696924567222595, 0.3194226026535034, -1.6294188499450684, 1.1000779867172241, 0.3027493357658386, -0.33169978857040405, 0.3734826147556305, 0.0941942036151886, -1.5713257789611816, 1.0335285663604736, -0.12083965539932251, -1.4671549797058105, -0.12235202640295029, -0.19406861066818237, -0.2006063014268875, -0.29184842109680176, 0.4606938362121582, -0.19156520068645477, -1.3798160552978516, 0.37361279129981995, 1.6120526790618896, 0.9505929350852966, 0.2615760564804077, -1.1621075868606567, -0.3379817008972168, -0.2234264463186264, -1.0351755619049072, 0.75633305311203, 0.5479499101638794, -0.16756248474121094, -0.27354076504707336, -0.3746376931667328, -0.25992071628570557, 1.091805338859558, -0.7455427646636963, 0.9382578134536743, -1.4920672178268433, -0.877411425113678, -0.15020781755447388, -0.5253186225891113, 0.15765009820461273, 0.07214213907718658, -0.8127281069755554], [0.5353701114654541, -0.012691710144281387, -0.4653611481189728, -0.05636956915259361, 0.7067572474479675, -1.0549274682998657, 1.4776049852371216, 0.3534959852695465, 1.5029850006103516, 0.24874159693717957, 1.0831037759780884, 0.6106804609298706, -0.09787004441022873, -0.5000023245811462, -0.175179123878479, -0.8423046469688416, -1.4327479600906372, 1.717811942100525, -1.8228517770767212, 3.4419193267822266, 0.6235738396644592, -0.5154930353164673, 1.2666568756103516, 1.1034293174743652, 0.5967704057693481, 1.7315254211425781, -0.8087376952171326, -0.11055590212345123, -0.3943220376968384, 0.47235172986984253, 2.2759480476379395, 0.18729960918426514, -0.37931591272354126, 1.154855489730835, 0.3196747303009033, -1.6143535375595093, 0.10586245357990265, 0.4101966321468353, 0.20143133401870728, -0.6603310704231262, -0.155209019780159, 1.9507170915603638, -0.7913764119148254, 0.9248966574668884, -1.1271177530288696, -1.1342324018478394, -1.0950958728790283, -1.8720018863677979, -0.8622272610664368, 0.20160619914531708, -1.2560944557189941, -0.0325055755674839, 1.436564564704895, 1.9204189777374268, 0.3037247657775879, -1.0908948183059692, -0.19809137284755707, -1.26502525806427, -0.29450926184654236, 0.8362985849380493, 0.8665430545806885, 2.3864188194274902, 0.8993275165557861, -0.23793116211891174, -0.8325265049934387, 1.7701900005340576, -0.19394366443157196, 0.6853975057601929, -1.824012041091919, -0.18446548283100128, -0.10539832711219788, -0.6562291383743286, 1.6994304656982422, 0.2284868061542511, 0.04742877930402756, 1.0939818620681763, 1.4293315410614014, 0.6532679200172424, 0.145745649933815, 1.0494476556777954, -0.49833834171295166, 1.6474140882492065, -1.2948927879333496, -0.4505750238895416, -0.41315338015556335, 1.4309804439544678, 0.6410325765609741, -0.16552358865737915, -0.22250863909721375, -0.24857935309410095, -0.7443114519119263, 0.2502758502960205, 0.5340932011604309, 0.641335129737854, 0.9561187028884888, -0.35349783301353455, -1.201524019241333, -1.590720534324646, 0.7039485573768616, -0.8644510507583618, -0.047405894845724106, -0.47564321756362915, -1.1998413801193237, 1.120992660522461, -0.325320303440094, 1.852133870124817, 0.7149413228034973, 1.7948704957962036, -0.47352999448776245, 0.13035796582698822, -0.574859619140625, 0.6912838220596313, 0.3213094174861908, -0.2504369914531708, 2.4293007850646973, 1.2945384979248047, 0.747407853603363, -0.7120099067687988, 0.17119690775871277, 0.613552987575531, 0.47451546788215637, 0.08378773182630539, -0.9914864897727966, -1.0320631265640259, -0.5431682467460632, 0.9666184782981873, -0.2550116777420044, 1.1072813272476196], [-0.6448929905891418, 0.7828817367553711, -1.1849228143692017, -1.2053104639053345, -0.4856303632259369, -0.30244556069374084, -0.02867378108203411, 0.9788264036178589, -0.03937281668186188, 0.7472718954086304, -0.519264817237854, -0.4614446461200714, 1.2398452758789062, 0.0671004056930542, 1.0748037099838257, 0.20793010294437408, -0.700326144695282, -0.7933351397514343, -1.1018399000167847, -0.13841569423675537, -0.8198668360710144, -1.358113408088684, -0.7938913702964783, -0.7037695050239563, 0.4677426218986511, -0.6018933653831482, -0.4956803023815155, 0.07316744327545166, -0.763656497001648, -0.38609346747398376, 1.2479326725006104, -2.1740636825561523, -0.4842061698436737, 1.3319059610366821, 0.10768426954746246, -0.14120884239673615, 0.6914048194885254, -2.3142240047454834, 2.0973265171051025, -0.5726127028465271, 1.3822749853134155, 1.1478395462036133, 0.40699154138565063, -0.1995403915643692, 0.24981465935707092, -0.2911628186702728, 2.0634710788726807, -2.4742162227630615, 0.07239188253879547, -0.08040347695350647, 0.03950827941298485, -0.9496254324913025, -0.6372781991958618, 1.5572137832641602, 0.31924688816070557, -0.47276532649993896, -0.5373669862747192, -1.9699254035949707, -1.538986086845398, -0.9127368927001953, -0.9772078990936279, -1.0224218368530273, 2.2028653621673584, 0.2074459195137024, 0.7223923206329346, -1.196351170539856, 0.7268471717834473, -0.6412039399147034, -0.3505837917327881, -2.0421080589294434, -1.4101057052612305, -0.4254196584224701, -1.094552755355835, -0.5712600350379944, -0.11049429327249527, 0.31969791650772095, -1.1059954166412354, 0.594788134098053, 0.4045921564102173, -0.37702515721321106, 0.280085027217865, 0.3664068281650543, 0.6153377294540405, 0.3720657527446747, 0.021872883662581444, -0.885430097579956, 0.33919841051101685, 0.6126805543899536, -0.8000258207321167, 0.09372954070568085, -0.8967371582984924, 0.8360729813575745, 0.024993455037474632, 0.23257142305374146, 1.0960443019866943, 0.23528869450092316, -0.38127854466438293, 0.27554935216903687, -2.0881762504577637, 0.18946564197540283, 0.023067016154527664, -0.18227575719356537, -0.3787931799888611, -1.9580439329147339, 0.2465221881866455, -0.0976574644446373, 1.481339693069458, 0.054157599806785583, -1.4947518110275269, -0.8157602548599243, 0.02440943941473961, -0.33923229575157166, 1.100665807723999, 0.8574692606925964, -0.38993731141090393, 0.3229348063468933, 0.5041584372520447, 0.2122940719127655, 2.9372849464416504, -0.997726321220398, 0.3824540674686432, -1.574846625328064, 0.8891780376434326, 0.5265229940414429, -0.6384699940681458, -1.144781470298767, -0.3236832618713379, 0.33461734652519226], [0.10404395312070847, 0.3850862383842468, 1.7130626440048218, -2.47902250289917, 2.869065046310425, -1.1526925563812256, -0.5645914673805237, -0.3210979998111725, -0.7832261323928833, -0.3907563388347626, -0.16288712620735168, 0.12723183631896973, -0.5750771760940552, 2.1127984523773193, -0.29475799202919006, 0.2634780704975128, 1.7450319528579712, -0.06906665116548538, 0.14501111209392548, -0.847507894039154, 0.5029759407043457, -0.6450351476669312, 0.07870995253324509, -0.4040810763835907, -0.9784350991249084, -0.08358141034841537, -1.7034112215042114, 0.798612117767334, 0.3019454777240753, -0.47316229343414307, 0.8931468725204468, -0.21898166835308075, 0.9905407428741455, -0.524653971195221, -0.08462990075349808, 0.9984298348426819, 0.15495634078979492, -1.3114814758300781, 0.7610007524490356, 0.08269336819648743, 1.0700567960739136, -0.5116013884544373, -0.10746614634990692, 0.18829210102558136, 0.9156109690666199, 0.07124551385641098, 1.003452181816101, 0.6239644885063171, -0.5891512632369995, -1.2216787338256836, -0.7825005054473877, -1.6869661808013916, -0.6969348192214966, -0.12801174819469452, -0.2027420848608017, 0.3706513047218323, 0.7629883885383606, 0.47670382261276245, 1.087573766708374, 0.5757049918174744, 1.0545557737350464, -1.1527444124221802, 0.48175176978111267, -0.01453462801873684, 1.6623117923736572, 0.7082376480102539, -0.7984172105789185, 1.084457278251648, 0.28616517782211304, -0.43641889095306396, 2.409287929534912, -0.5406099557876587, 0.0686604231595993, 0.019071847200393677, 0.6195600032806396, 1.1974592208862305, -1.9259718656539917, -0.13593502342700958, -1.2116848230361938, 1.0819722414016724, -0.22854982316493988, -0.6174112558364868, 0.17028270661830902, 0.15436352789402008, -0.16685418784618378, 0.4173864424228668, -0.1289328932762146, 0.08614575117826462, 1.0938140153884888, -1.6611486673355103, -1.066759705543518, 0.46192651987075806, 0.3993843197822571, 0.5922939777374268, -1.2037684917449951, -1.3797147274017334, 0.085030697286129, -0.5324466228485107, -0.6497278809547424, -0.17587685585021973, 1.3608983755111694, 1.6220372915267944, -0.4663587510585785, 1.025145411491394, -1.1344166994094849, -0.7953159213066101, -0.2342146933078766, 0.8802027106285095, 0.3218194246292114, 0.9448336958885193, 1.2232125997543335, 0.20048780739307404, 0.09538068622350693, -1.0657591819763184, 0.9674454927444458, -0.5937132835388184, 0.8239743113517761, -0.5574004054069519, 0.3899206519126892, -0.852022111415863, -0.5627543330192566, -0.03445292264223099, 0.7599979043006897, 0.23881781101226807, 0.3406856656074524, 0.899017333984375, 1.190713882446289, -0.3224222958087921], [-0.6770466566085815, 1.8051036596298218, -0.702582061290741, 0.3227527141571045, 1.2560203075408936, 1.0805974006652832, -0.40254974365234375, -1.7204294204711914, 1.3508330583572388, 0.5921136140823364, 0.12227474898099899, -0.04152745380997658, -0.4089713990688324, 0.3438667356967926, -0.5540446639060974, 0.7200673222541809, -0.9962966442108154, 0.6177312135696411, -0.08760453015565872, -1.0323212146759033, 0.8338677883148193, -2.080892562866211, -0.30513256788253784, 0.8854455351829529, 0.3756054639816284, -0.00041799808968789876, -1.1590689420700073, 0.49637171626091003, -0.6290345788002014, -0.27098536491394043, -1.3223488330841064, -0.20669275522232056, 1.2488166093826294, -1.6539734601974487, 0.558133065700531, -0.3652765154838562, 0.38883018493652344, 1.6701650619506836, 1.2450774908065796, -0.6881817579269409, -0.8281333446502686, -1.267958402633667, 1.4835323095321655, 0.5429552793502808, 1.0791373252868652, 1.5772796869277954, 1.3861788511276245, 1.3424991369247437, -0.8230985999107361, -0.05799425393342972, -0.7969516515731812, 0.5581783056259155, -1.5612274408340454, 1.2396690845489502, 0.5939271450042725, -0.42117705941200256, 0.8892526626586914, 0.7596536874771118, -2.07533860206604, 1.9257493019104004, -0.03344366326928139, -0.08568631857633591, -0.11273416876792908, -1.268013834953308, -1.0486652851104736, 0.4951764941215515, 1.4573931694030762, -0.9317841529846191, 0.7789756059646606, 0.7413256764411926, -0.008234427310526371, 0.84396892786026, 0.5543785691261292, 1.4211241006851196, 0.7230109572410583, 2.493659496307373, 1.6434077024459839, 0.15076619386672974, -0.7352361083030701, -1.0852309465408325, 1.3140509128570557, -1.3699380159378052, 0.2318020612001419, -0.48571258783340454, -0.5807296633720398, -0.06159858778119087, 0.29845404624938965, -1.6573954820632935, -0.5348536372184753, 1.3404597043991089, 0.3114983141422272, 0.9375129342079163, -1.8553158044815063, 0.2186039537191391, 0.87811678647995, -1.9981420040130615, 0.11636197566986084, 0.26834577322006226, -1.6990947723388672, -0.14047230780124664, 0.3089837431907654, 1.083013892173767, -0.2208707183599472, -0.8969720602035522, -0.621696949005127, 0.5892142057418823, 0.12786957621574402, 1.004621982574463, 0.9702352285385132, 2.116935968399048, -0.2258785367012024, -0.08197899907827377, -1.8962301015853882, 1.2483124732971191, -2.4636170864105225, -0.4196501672267914, -0.8487778306007385, -0.23367564380168915, 2.27984356880188, -0.11972519010305405, -2.7791409492492676, -0.7219687104225159, -0.40834319591522217, 1.2854636907577515, 0.10485587269067764, 0.5122151970863342, 0.12687909603118896, 0.9203453660011292], [-1.2187178134918213, -0.44202885031700134, -1.2417755126953125, 0.04634272679686546, 1.341525673866272, 0.4579513967037201, -0.8163158893585205, 0.7517670392990112, 0.013285105116665363, -1.1510106325149536, 0.4662136733531952, -0.3890775740146637, -0.02213357575237751, 0.3236486315727234, 0.6987125277519226, -0.7013025879859924, 1.929009199142456, -0.46953076124191284, 1.6494590044021606, 0.4561741054058075, -0.15409298241138458, 0.9932317137718201, 0.5534648299217224, -1.0932104587554932, -2.6440317630767822, -0.7074271440505981, -0.3670015335083008, -0.23730166256427765, 0.24259422719478607, 1.591630220413208, -1.4703487157821655, -0.7626565098762512, -2.072941541671753, -1.7819883823394775, 1.842851161956787, -2.2059147357940674, -1.0199027061462402, 0.26982381939888, 0.2992798984050751, 0.19231413304805756, -1.3594335317611694, 0.9580864906311035, -1.0148881673812866, -1.0482994318008423, -0.759770929813385, -0.8641301989555359, -0.7542495131492615, 1.634225845336914, 0.4320096969604492, 0.015289179049432278, 1.3420913219451904, -0.3395128846168518, -0.36552438139915466, 0.19212590157985687, -1.0119879245758057, -0.8441523313522339, 0.4150727391242981, 0.24288862943649292, 0.28045088052749634, -1.2859901189804077, -0.46032610535621643, -1.9154951572418213, -0.7161369919776917, 0.2811446487903595, -0.9312044382095337, -0.3137510418891907, -0.3977256715297699, 0.7073251605033875, 0.18740840256214142, 1.1875790357589722, -0.8545235991477966, 1.357274055480957, 0.16425639390945435, -0.6646185517311096, 1.5321524143218994, 1.1686254739761353, -0.8043721914291382, 0.30123209953308105, -0.12317555397748947, -1.510521411895752, -0.4616767168045044, -0.6756590604782104, 0.803834080696106, -0.4581809937953949, -0.3505575358867645, -0.6007758975028992, -0.051185157150030136, 0.9148598313331604, -1.188822865486145, 2.021074056625366, 0.5567216277122498, -1.2325869798660278, 0.7085700631141663, -0.9889108538627625, -2.4062867164611816, 0.43796804547309875, 0.22535353899002075, 1.2437373399734497, -1.427722454071045, -1.4014133214950562, -0.3392350375652313, 2.4193263053894043, -0.557551383972168, -0.1313815414905548, 0.5717676281929016, -0.1875835806131363, -1.0852928161621094, -0.02975068800151348, 2.4210221767425537, -1.0578029155731201, 0.884653627872467, -0.9516093134880066, 1.0772184133529663, 0.10251422971487045, -0.1276681274175644, 1.07106614112854, 2.2472050189971924, -1.1663501262664795, -0.7009865641593933, -0.16000725328922272, -0.4089896082878113, -0.3340246379375458, -1.5643247365951538, -1.6768555641174316, 0.10627442598342896, 0.7118872404098511, -0.21676819026470184, 1.45785391330719], [-2.6158435344696045, 0.1141253113746643, -0.2705560326576233, 1.191475749015808, 0.28474611043930054, 1.3056628704071045, -0.7390174269676208, -0.4440285265445709, -0.32966119050979614, -0.6692350506782532, 0.20311972498893738, -1.290342926979065, -0.1042570173740387, 1.1022356748580933, 1.1194323301315308, 0.016411835327744484, 0.9444718956947327, 0.3436449468135834, -1.4430615901947021, -0.7519723773002625, 0.897229015827179, -0.025139853358268738, -0.07673311233520508, -0.5219045281410217, 0.8570784330368042, -0.010053793899714947, -0.8433873057365417, 0.06016138568520546, -1.144208550453186, 0.11319153755903244, 0.04144900664687157, -1.0934021472930908, 0.19165945053100586, -0.31097137928009033, 0.463007390499115, -0.8773528337478638, 0.25877174735069275, -0.3056641221046448, -0.31121718883514404, 0.3108486831188202, 0.07945465296506882, 0.35278093814849854, 0.17849847674369812, -0.937484085559845, -1.092481255531311, -1.323617935180664, 0.9110068082809448, -0.2789793014526367, -0.735991895198822, 0.1387806236743927, 0.3947572708129883, 0.1018834263086319, -0.11071839928627014, 0.4092956781387329, -0.011348090134561062, -1.4730675220489502, 0.37726110219955444, -0.38755184412002563, -0.21701186895370483, -0.6579998731613159, -0.4895702600479126, -0.2989248037338257, -0.8807732462882996, -0.5291529893875122, 0.9226421117782593, 0.05609304830431938, 1.106062650680542, 1.7570440769195557, -1.2597724199295044, -0.8985150456428528, -0.11395715922117233, -0.05247143283486366, -0.4086087644100189, 0.02399991638958454, -1.8778421878814697, -0.8919371962547302, -0.5887132287025452, 0.884565532207489, -0.6950272917747498, 0.7731845378875732, 0.1574029177427292, 0.5671020746231079, 2.11812162399292, -0.028140820562839508, -0.44150981307029724, 0.9189707040786743, 0.4645043611526489, 1.4508788585662842, -1.3376542329788208, 0.176571786403656, 0.27558910846710205, 0.19155295193195343, -0.11091272532939911, -2.3469367027282715, 2.24043869972229, 0.6768949031829834, -0.21205033361911774, -0.46188169717788696, -0.440269410610199, 0.5336020588874817, 0.6422078609466553, -2.00974440574646, -0.5121827721595764, -0.05321747809648514, 0.9778660535812378, 1.2305952310562134, -1.5171246528625488, -0.10726901888847351, 0.285616934299469, 0.3684946298599243, 0.6932882070541382, 0.2070801705121994, -0.3894582688808441, -0.3864879608154297, -1.9241105318069458, 0.11928002536296844, -0.19798991084098816, -1.0535975694656372, -0.1207834780216217, -0.799866795539856, -0.0032044770196080208, 1.6023011207580566, 0.3434586822986603, -1.4450868368148804, 0.29201555252075195, -0.20602373778820038, -0.05448390543460846, 1.2550227642059326], [-0.059847719967365265, 0.6205161809921265, -0.4129248261451721, 1.2150133848190308, 1.6375315189361572, -1.1577855348587036, -0.6754137277603149, 0.5179288983345032, -1.1852084398269653, -0.0008872862672433257, -1.0071394443511963, -0.40954023599624634, -0.7976044416427612, -2.2049827575683594, 0.31807202100753784, 1.2590645551681519, 0.42096471786499023, -0.08648591488599777, 0.459638386964798, 0.8155697584152222, -0.7417249083518982, -0.23779986798763275, 0.1899319440126419, -0.3063620328903198, -0.19447241723537445, -0.7278440594673157, 0.021586978808045387, -2.0277199745178223, -0.40048378705978394, -2.236327648162842, 0.3985079526901245, -0.21129758656024933, -0.45135563611984253, -0.5093441009521484, 0.4826491177082062, -0.25532013177871704, 0.34352272748947144, -0.8297752737998962, 0.20523443818092346, 0.33328184485435486, 0.2501283884048462, -0.7505201101303101, 0.08776205778121948, -1.1962685585021973, 1.6246836185455322, 1.0549976825714111, 0.13058045506477356, 0.3376404345035553, -0.8533107042312622, -1.038838267326355, -0.5234265923500061, -0.18751712143421173, 0.4398159086704254, 2.1073484420776367, 0.9435060620307922, 0.7062227129936218, -0.5638723969459534, -0.3291725218296051, -1.0392251014709473, -1.1534892320632935, 0.8460588455200195, 0.7076718807220459, -1.8460873365402222, -1.2394167184829712, -0.5707194209098816, -0.8669734597206116, 0.24190694093704224, -0.8737456202507019, 0.17197465896606445, 0.6808804869651794, -0.1413027048110962, -0.22226539254188538, -1.0242388248443604, 0.33089229464530945, 1.390726923942566, -1.314354658126831, 0.19286946952342987, 0.8022393584251404, 1.3586678504943848, 1.1000258922576904, 0.3557104170322418, -0.42528894543647766, -0.8001049757003784, 0.3130147457122803, 0.034491486847400665, -0.18639366328716278, -0.7961927652359009, -2.145097255706787, -1.384517788887024, -0.9597668051719666, -1.1245390176773071, -0.7295556664466858, 0.062008120119571686, -1.2767252922058105, 1.755023717880249, -1.9492971897125244, 0.8832931518554688, 0.5229344367980957, -1.1560250520706177, 2.271343469619751, -0.41381940245628357, -2.263108015060425, -0.4092719554901123, 1.0363459587097168, 0.6402897834777832, 1.8861167430877686, 0.9900254011154175, 0.7586880922317505, -0.8099092245101929, -1.2291158437728882, 1.4626007080078125, -0.1671942174434662, 0.34236788749694824, -1.5164741277694702, 1.096719741821289, -1.0409135818481445, -0.9070073962211609, -0.259469211101532, 0.13410601019859314, -0.5645686984062195, 0.5967978239059448, 1.649841547012329, -2.0602433681488037, 0.3772052824497223, 3.0742545127868652, 0.4764154553413391, -0.09314277768135071, 1.4133163690567017], [0.2346336990594864, 1.9327391386032104, -0.7478625774383545, -0.568943440914154, -1.480857253074646, 0.011847829446196556, -0.29085978865623474, 0.1281331181526184, 1.140389323234558, 1.0788344144821167, -0.2116331309080124, -0.6297260522842407, 1.257180094718933, 0.6303170919418335, 0.5316563844680786, -0.2314174473285675, 1.155576467514038, 0.8541262745857239, 0.8648840188980103, -0.9393776655197144, -0.67002934217453, -0.5324909687042236, 0.18487872183322906, 0.9078885316848755, -0.20864760875701904, -0.9535794854164124, -0.4968765676021576, -0.43620309233665466, -1.0516325235366821, 0.3633269965648651, 0.3009876012802124, 1.8276221752166748, -0.2304142266511917, 0.19045698642730713, -0.377477765083313, 0.12604159116744995, 0.3060070872306824, 0.6923436522483826, 0.5416049957275391, -0.7506083250045776, 0.3996361494064331, -0.4776748716831207, 0.23703499138355255, 0.31869545578956604, 1.1581473350524902, -0.42662590742111206, -0.38900429010391235, 0.6558471322059631, -1.0010502338409424, 0.21525363624095917, -1.3117995262145996, -1.1373859643936157, 0.2575671374797821, 2.2569079399108887, -1.8096977472305298, 0.6256672739982605, 2.8168158531188965, -0.4539409875869751, -1.1102765798568726, -2.2321155071258545, 0.373675674200058, 1.0839893817901611, 1.6458933353424072, -0.019087884575128555, 0.680336594581604, 1.1723183393478394, 0.7102546095848083, 1.261893630027771, -1.0676015615463257, -0.8098644018173218, -0.26293423771858215, -0.652695894241333, 0.7876121401786804, 0.6561001539230347, 1.212000846862793, 0.09169898927211761, 1.0977938175201416, 0.5417161583900452, -0.10138393193483353, 0.01903916709125042, -1.2921826839447021, -1.870725393295288, 0.25729990005493164, 1.4026869535446167, -0.6112109422683716, -1.1128870248794556, 2.175509452819824, 0.45374664664268494, -0.8890997767448425, -0.40592288970947266, -0.6132056713104248, -0.6498266458511353, -1.4312008619308472, 1.824373722076416, 0.9386568069458008, 0.07099085301160812, 0.10298477113246918, -0.6334248781204224, 0.7795170545578003, -0.704034686088562, 0.3436775505542755, 0.06730353087186813, -0.36322638392448425, 0.7649755477905273, -0.6068649291992188, -0.9117646813392639, -0.015538318082690239, 0.01811235584318638, 2.662024736404419, 1.141591191291809, -0.9952392578125, -1.0741957426071167, 0.1857559084892273, 0.33382648229599, 1.2250252962112427, 0.4908706545829773, -0.2688627541065216, 0.6045913696289062, -0.3959903120994568, -0.8269343972206116, 0.9153565764427185, -0.5517545938491821, -0.44276612997055054, 0.823866605758667, -0.8836714029312134, 0.5877604484558105, 1.2623111009597778, -0.935555636882782], [1.585586428642273, -0.35090237855911255, -0.05227169394493103, 0.9055235981941223, 1.0963380336761475, -0.5295907855033875, -0.8424914479255676, -0.37012794613838196, -0.7148021459579468, -0.7213386297225952, 0.5734818577766418, 1.9312673807144165, -0.23583443462848663, -0.38878732919692993, 2.188711166381836, -1.0518052577972412, 0.5183076858520508, -0.38103410601615906, 0.24544332921504974, -0.7007955312728882, 0.2984616160392761, -0.46014097332954407, -1.1131856441497803, -1.2289931774139404, 0.8660207390785217, 2.176515579223633, 0.5105005502700806, -0.29261913895606995, 0.48033639788627625, 1.5159410238265991, 0.15005619823932648, 1.0071687698364258, -0.2795373201370239, -0.9724514484405518, 0.40567126870155334, -1.8396859169006348, -0.997240424156189, 0.034541357308626175, -2.258657932281494, -2.3481578826904297, -0.32729196548461914, -0.8065522313117981, 0.11367925256490707, -0.20907384157180786, -0.6077299118041992, 1.3650140762329102, 1.2310477495193481, 0.07424461096525192, 1.5056238174438477, 0.3114829957485199, 0.6940680146217346, 0.350321888923645, 0.4707205295562744, -0.2690844237804413, -0.2365391105413437, 0.05463311821222305, 0.5492784976959229, 1.886427879333496, 1.391731858253479, 0.005103515461087227, -0.08364060521125793, -1.1342424154281616, -0.849655032157898, -1.188201665878296, 0.02043653093278408, 0.7828440070152283, -1.0439225435256958, 1.5140830278396606, -0.7904909253120422, 0.8054107427597046, 1.5522494316101074, -1.1078599691390991, 2.012098789215088, 0.6539416909217834, 0.9589526057243347, 0.21862515807151794, 1.5456907749176025, 0.26212078332901, 2.1345326900482178, -0.5195248126983643, -0.8828580975532532, 1.1271576881408691, -0.049451593309640884, 2.1416139602661133, -0.8447321057319641, -1.0183138847351074, 0.05976644530892372, -0.21706530451774597, -0.051052406430244446, 0.8273817896842957, -1.0221704244613647, -0.7286556363105774, -1.3005369901657104, 0.3707429766654968, 0.5027804374694824, -1.9336857795715332, -0.5088558197021484, 0.17284274101257324, 1.6375834941864014, 0.863746702671051, 0.44063785672187805, -0.8067634701728821, -0.55415278673172, -0.36855512857437134, 0.676840603351593, 2.031519889831543, -0.9152410626411438, 0.12078574299812317, -0.7571761608123779, 1.1170976161956787, 0.5470037460327148, 0.6679972410202026, -0.8529043793678284, -0.29216766357421875, 0.45746585726737976, 0.7088279128074646, -0.6839810609817505, -0.16144968569278717, 1.6823369264602661, -0.949944019317627, -0.36553823947906494, -0.06502699106931686, 0.5160516500473022, 0.18984058499336243, 0.5436108708381653, 0.812192976474762, -0.9674621820449829, 0.22998277842998505], [1.2328518629074097, -0.7829192280769348, 0.6145099401473999, -0.9402341842651367, -0.9093976020812988, -0.1663333773612976, -1.0441428422927856, -0.058147188276052475, 0.24497883021831512, -1.173237919807434, 0.016397371888160706, 0.6039869785308838, -0.9998157024383545, 0.8380177617073059, 0.6275432109832764, -1.6242172718048096, 1.0020097494125366, -1.2400001287460327, 0.4958690106868744, -1.3648947477340698, 0.1144089624285698, -1.5307798385620117, -1.6166281700134277, 0.21220754086971283, 0.27813369035720825, -1.9928693771362305, 1.0629278421401978, 0.3995806574821472, -0.44964879751205444, 0.13211920857429504, 0.06387435644865036, 0.5214953422546387, -0.3671363294124603, 1.1790708303451538, -0.9955450296401978, 2.3497064113616943, 0.025057006627321243, -0.2650187909603119, 0.6545522212982178, -0.05405480042099953, -0.012846329249441624, 1.1322838068008423, 1.6121848821640015, 1.3144147396087646, 0.1691461205482483, -0.023388994857668877, -0.6833286285400391, 0.7688048481941223, 0.4195597767829895, 0.8545041084289551, 2.063606023788452, -1.0722119808197021, -0.5051915049552917, 0.4213438928127289, -0.3310415744781494, -0.11069173365831375, 1.1754907369613647, -0.6374839544296265, -1.3004276752471924, 0.9418479204177856, -0.4904184937477112, -0.2826816141605377, 0.0646372064948082, -0.8285999894142151, 0.0249467846006155, -1.2207101583480835, 0.5180232524871826, 1.1439720392227173, 1.6313838958740234, 2.7076265811920166, -1.1193578243255615, 0.19237282872200012, 1.7876490354537964, -0.45995432138442993, 0.8993526697158813, -1.2911444902420044, -0.8892064094543457, 1.0668931007385254, 1.6213734149932861, -0.6460960507392883, -1.1351635456085205, -0.9749360680580139, 1.0625144243240356, -0.6868239641189575, -1.7731035947799683, 0.47676944732666016, 0.4237326383590698, -0.5120992660522461, -1.6898640394210815, 2.0874240398406982, 0.2949085533618927, -0.8340200781822205, 1.8991419076919556, 0.3598354756832123, -0.7149474620819092, -0.9580086469650269, 1.4361652135849, -0.9743090867996216, 0.3343872129917145, -1.6558012962341309, 0.13953208923339844, -0.5795975923538208, 0.9184026122093201, 0.811393141746521, -0.7423930168151855, 0.11611910164356232, -1.4644877910614014, 1.4086458683013916, 0.7848925590515137, 0.2932073771953583, 0.9843947887420654, 0.5907356142997742, 0.09233549237251282, 0.1134830117225647, -0.41573652625083923, 3.259340763092041, -0.5208158493041992, -2.2646143436431885, -0.37897494435310364, 0.6413764953613281, -0.14331211149692535, -1.2286158800125122, 0.23104117810726166, -0.9539704918861389, -1.6042842864990234, -0.37544944882392883, 0.8983903527259827, 0.11680705100297928], [-0.21537791192531586, -0.24908855557441711, -0.5288270115852356, 1.2257276773452759, -1.6581885814666748, -0.9743111729621887, -0.0668027400970459, 0.5861430764198303, 0.30099865794181824, -0.5684199929237366, -0.4906638562679291, -2.5178465843200684, 0.018748881295323372, -0.14239534735679626, 0.3183229863643646, -0.0594254732131958, -1.179267168045044, -1.355997085571289, -1.0131723880767822, -0.30771705508232117, 2.8946216106414795, -0.05299459397792816, -2.686392307281494, 0.10462722927331924, 2.4518070220947266, -2.2745749950408936, 1.0108304023742676, -0.5727227926254272, 1.0672168731689453, -0.9213963747024536, 0.5473336577415466, 0.4474506676197052, -0.7877812385559082, -0.5010673999786377, 0.3400498926639557, 0.20703552663326263, 0.14906756579875946, -0.34476161003112793, 0.10789866745471954, -0.05312671884894371, 1.5004535913467407, 1.2397403717041016, -1.8511111736297607, 1.940181016921997, -0.1257268488407135, -0.32941025495529175, -0.673433244228363, -0.22307121753692627, -1.0760761499404907, -0.42852941155433655, 0.6525692939758301, -1.1988670825958252, 0.5788227915763855, -0.997555136680603, -0.4964514970779419, -0.5229364633560181, -0.8150097131729126, 0.22052176296710968, -2.2319886684417725, 1.6264233589172363, -0.09750867635011673, -1.2097355127334595, 0.014646872878074646, -0.8696702122688293, -1.3773561716079712, -0.7215336561203003, -0.17446374893188477, -1.2053486108779907, 0.40886491537094116, -1.07378351688385, -1.080857753753662, -1.2396647930145264, 0.4455191493034363, 0.22254954278469086, 0.40081265568733215, -0.9872320294380188, -0.5286598205566406, -0.34179210662841797, -0.7212871313095093, -0.3281656503677368, -0.1191454529762268, 0.32593533396720886, 1.512208104133606, 0.7940516471862793, -0.3592696189880371, 1.2099480628967285, -1.5514438152313232, 0.8044108748435974, -0.9372342228889465, -2.292888641357422, -0.5562186241149902, 0.4441279470920563, -0.022254828363656998, -0.824822723865509, 0.09060714393854141, 0.0325821153819561, 1.2776751518249512, -0.7786887288093567, -1.9904390573501587, -0.5113791227340698, -0.012715181335806847, -0.8703904747962952, -0.23314924538135529, -0.47022974491119385, 0.3645261228084564, -0.5601003170013428, -0.7819127440452576, 0.38213780522346497, -0.7457538843154907, 1.1922075748443604, -0.015130911953747272, 0.6313763856887817, -0.8843415975570679, -0.1668294221162796, 0.5693690180778503, -0.6424914002418518, -0.4118642807006836, -0.10656608641147614, 0.8989484310150146, 0.2918268144130707, 0.2123689502477646, -0.32903367280960083, 1.4942002296447754, 1.0211586952209473, 0.8989136219024658, -0.39458832144737244, -0.6794130802154541, -0.5565900802612305], [-0.674240231513977, -1.0132516622543335, 1.3773596286773682, 0.3214305341243744, 1.1470907926559448, -0.5635467171669006, 1.2863346338272095, 0.9644817113876343, -1.1432746648788452, 0.12738487124443054, 0.07356376945972443, 1.82882821559906, -2.4335551261901855, -1.0036579370498657, 1.434424638748169, -2.195613145828247, 0.6519201993942261, -1.3003889322280884, -0.08330120146274567, 0.5506670475006104, -0.31121477484703064, 1.068719744682312, -0.24836860597133636, 0.9289913773536682, 1.8380966186523438, 0.2727498412132263, 0.5303915739059448, 0.9247944355010986, -1.4066400527954102, -1.5304739475250244, 0.738508403301239, 1.4150409698486328, -0.08765190094709396, -0.3919057846069336, -0.702613890171051, -0.8816413879394531, -0.8833340406417847, 2.383774518966675, 1.5275846719741821, 1.4257864952087402, -0.4043821692466736, 0.1232648715376854, 1.3586117029190063, 2.0384931564331055, 0.5675883889198303, -0.7002530694007874, 0.2938421666622162, 0.059850938618183136, 1.9099467992782593, 1.2082538604736328, -0.6582364439964294, 0.34674128890037537, 1.340933918952942, -1.3577873706817627, -1.7773535251617432, 0.44317787885665894, -0.3518420159816742, -0.23654450476169586, -1.1536070108413696, -1.782918930053711, -0.7730772495269775, -0.2178792655467987, -0.8659269213676453, -0.21606627106666565, -1.0952510833740234, 1.0456191301345825, 0.07281282544136047, 0.35079649090766907, 0.6296545267105103, 0.8821863532066345, 0.8136013746261597, 0.0781627893447876, -0.38341131806373596, 0.6880490183830261, -1.1630607843399048, 1.20192289352417, 0.4006898105144501, 0.29809704422950745, 0.3227187991142273, 0.6387341022491455, 0.8710342049598694, 0.2147177755832672, -0.14911910891532898, 0.35088104009628296, -1.5215173959732056, -0.44979169964790344, 0.18100093305110931, 0.732603907585144, 0.7068331837654114, -0.47808972001075745, -0.44050902128219604, 0.32215702533721924, -0.05359762907028198, 0.4740714430809021, 1.8658568859100342, 1.0234483480453491, 0.3999754786491394, 0.8357415795326233, 0.5088079571723938, -1.279765248298645, 1.411785364151001, -0.5089505910873413, 0.07592131942510605, 1.6357166767120361, 1.0210610628128052, 0.2729234993457794, -1.0938607454299927, 1.397904396057129, 1.1362016201019287, -0.4976409673690796, 1.2427626848220825, -0.37409260869026184, -0.4607597589492798, 1.0808969736099243, -0.782856822013855, 0.8716723322868347, 0.16421882808208466, -1.5775182247161865, 0.48496851325035095, -0.8720828890800476, 0.7563187479972839, -1.1952658891677856, -0.2818911373615265, -0.2660232186317444, 2.126814842224121, -1.0510108470916748, -0.2794557213783264, 1.3939481973648071], [0.4064972400665283, 1.0650107860565186, 0.3692023456096649, 1.4205464124679565, -0.16379070281982422, 0.1115143820643425, -1.8203299045562744, 0.3428545892238617, 0.537078857421875, 1.2404872179031372, -0.590226411819458, 1.618489146232605, -1.4768621921539307, -1.1909009218215942, 1.0111278295516968, 2.3244850635528564, 0.7195126414299011, -0.6291541457176208, -1.0688797235488892, 0.3571288287639618, -0.13361135125160217, -0.932437002658844, 0.10620126873254776, 1.129898190498352, 2.503598928451538, -0.9807766675949097, -0.2348024994134903, -0.1970888376235962, -0.2978806793689728, -0.9906725883483887, 0.09026689827442169, 1.3105087280273438, 0.1661110520362854, 0.26711076498031616, 0.9940507411956787, 1.2549811601638794, -0.2629713714122772, 1.9109970331192017, 1.3449536561965942, 1.915321707725525, 1.1915390491485596, -0.4256631135940552, -1.3258485794067383, 2.3538315296173096, -0.3095678687095642, -0.2197503000497818, -1.2816038131713867, 0.3012407422065735, 0.5460547804832458, 0.3689385652542114, -0.322925329208374, 0.09962044656276703, -0.34268414974212646, 0.7492766380310059, -0.23745019733905792, -1.2968169450759888, 0.5966785550117493, -0.24573799967765808, -1.7880905866622925, -1.4660347700119019, -0.989502489566803, -1.3133127689361572, -0.850055992603302, -0.7020485997200012, 0.016325082629919052, -1.1690057516098022, -1.0602318048477173, 0.9259688258171082, 1.648675799369812, 0.5152929425239563, 0.9167160391807556, -1.8847631216049194, -0.3195393979549408, 0.1827744096517563, 1.0095155239105225, 0.9443264007568359, -0.8213995695114136, 1.0474179983139038, -0.6912565231323242, -0.597322940826416, 0.2257184535264969, -0.8344758749008179, -0.007550508249551058, -0.1574755758047104, -1.0412839651107788, 0.6528223156929016, -0.7884113788604736, 1.287585973739624, 0.21311762928962708, 2.6432459354400635, -1.4570099115371704, -1.5598610639572144, -0.05972305312752724, -0.695846676826477, -1.011271595954895, -0.6357024908065796, 0.023808740079402924, -0.024375829845666885, 0.04127219319343567, 1.5205636024475098, 0.5127003192901611, -1.4300165176391602, 0.9752975702285767, 0.22795642912387848, -0.5184799432754517, -0.6237280368804932, -0.8222517967224121, -0.5707696080207825, 1.8836729526519775, -0.4239179491996765, -0.3013969957828522, 0.03454264625906944, 1.5871862173080444, 2.53175687789917, 0.6238283514976501, -0.9414236545562744, -0.21650950610637665, -0.6464626789093018, -0.2173851728439331, 0.09243492037057877, 0.39489489793777466, 0.8882998824119568, -1.576674461364746, 1.643601655960083, -1.1809911727905273, -0.0006683069514110684, 0.9196714162826538, 1.1069419384002686], [-1.0367158651351929, 0.23534540832042694, -0.4707544147968292, -0.7344451546669006, 0.9733754396438599, 2.0529568195343018, 0.7940793633460999, 0.13777141273021698, 0.7904369235038757, -1.2150413990020752, 0.38816773891448975, -1.2180440425872803, 1.6376887559890747, -0.9327693581581116, 0.3434731364250183, 1.149548053741455, 0.1271767020225525, -0.7022087574005127, -0.03760038688778877, -0.7043320536613464, -0.6888239979743958, -1.3527979850769043, -0.6697682738304138, 0.6248807907104492, -0.7660486102104187, 0.101980060338974, -0.6748239994049072, -0.7274016737937927, 0.05581360682845116, -0.27062562108039856, -1.1373915672302246, -0.5469127297401428, -0.3517482280731201, -0.5415732860565186, -0.4434482753276825, -1.0414923429489136, 0.1621556431055069, 0.23425038158893585, 1.288338541984558, -1.1451590061187744, -0.276770681142807, -0.7738783359527588, -1.1729782819747925, 0.5173197984695435, -1.9139066934585571, -0.06168169528245926, 0.7853729724884033, 0.8895204663276672, -0.1110558956861496, -0.9756402969360352, 1.0853958129882812, -2.011212110519409, -0.21926145255565643, -0.8550115823745728, -1.56882905960083, 0.5873135328292847, 1.0819330215454102, 0.6041972637176514, 0.7173968553543091, -0.5560500621795654, 1.4884790182113647, -0.9170870184898376, 1.9360002279281616, -0.5104759335517883, 0.6262492537498474, -0.8557541370391846, -1.1197091341018677, 1.4796913862228394, 0.8513739705085754, -1.2482695579528809, -0.8662524819374084, 0.3583441376686096, -1.3271969556808472, -0.3022399842739105, -0.6537869572639465, -3.0941877365112305, -0.2992187440395355, 0.21759232878684998, 0.8569901585578918, -0.7570238709449768, 0.6967688798904419, 0.5315525531768799, -0.2818375527858734, -2.1168372631073, 0.6231086254119873, 0.6019924283027649, 1.1323729753494263, -0.37843403220176697, 0.034506458789110184, -0.4920010566711426, 0.5198871493339539, 0.8917905688285828, -1.2399506568908691, 0.02494954690337181, -1.2498401403427124, -1.9315195083618164, -0.607073187828064, 0.5484729409217834, -1.3981683254241943, 0.6506755352020264, 0.3073924481868744, -0.3888128697872162, 1.890377163887024, 0.7949628233909607, -0.2095613181591034, 0.2025107741355896, 1.1263389587402344, 0.23423363268375397, 0.05819429084658623, 0.42811280488967896, 0.9635310173034668, 0.20764988660812378, 0.9951127171516418, -1.0387802124023438, 0.770408570766449, 1.3419196605682373, 0.20675712823867798, -0.33616864681243896, 1.4151878356933594, 1.2380620241165161, -1.0207464694976807, -2.124865770339966, -0.9548389911651611, -0.1481345146894455, -0.06210608035326004, -0.7708704471588135, 0.1355820745229721, 1.3567891120910645], [-1.3289790153503418, -0.5736090540885925, 1.174160361289978, 1.186328649520874, -0.32595863938331604, -0.7298804521560669, 0.12167558819055557, 1.3301177024841309, -1.1832376718521118, -0.056204285472631454, 1.0598576068878174, 0.5492715239524841, -1.1610760688781738, -1.4986132383346558, -0.9224693775177002, -2.060547351837158, 0.9428311586380005, 0.7329380512237549, 0.531619668006897, 0.027045994997024536, -0.09203799068927765, -1.5837918519973755, 0.08621682226657867, -1.329519510269165, -1.2113689184188843, -0.8123772144317627, -0.8091709017753601, 0.21276433765888214, 0.42707201838493347, 0.3666478097438812, -0.8774981498718262, -0.027498023584485054, -1.2172456979751587, -0.5618786215782166, 0.9673067927360535, 1.805688500404358, 0.891081690788269, 0.08739425241947174, 0.0661264955997467, -0.5204229950904846, 0.561225414276123, -1.798298716545105, -0.5165286660194397, -0.033067379146814346, 1.1876527070999146, 1.5278266668319702, -1.6177785396575928, 0.5576158165931702, -1.3685944080352783, 0.6647096276283264, 1.1386878490447998, -0.10693083703517914, 0.9142453074455261, -1.2168912887573242, 1.3218094110488892, -0.9225838780403137, 1.0931733846664429, -0.1468620002269745, -0.23457179963588715, 1.223257303237915, 2.0170750617980957, -0.3258334994316101, 0.6568962335586548, 0.29868462681770325, 0.7931969165802002, -0.659874439239502, 0.34920671582221985, -0.1476467251777649, -0.6590886116027832, -0.9898090362548828, -1.4030755758285522, -1.680986762046814, 0.3930933177471161, 0.17508111894130707, -0.31630632281303406, -1.1937578916549683, -0.8331581950187683, -1.8689988851547241, 0.20060038566589355, -0.25838756561279297, -1.8650027513504028, 1.5683873891830444, -1.7321921586990356, 0.6529525518417358, 1.349269151687622, 0.7381157279014587, -0.13602399826049805, 1.9783878326416016, -0.5599006414413452, 0.4952446222305298, -1.2251043319702148, -0.7056388854980469, 0.6308253407478333, 1.3728965520858765, 0.6663592457771301, -0.6689398884773254, -1.6983799934387207, 0.28480958938598633, 1.2992147207260132, 0.40382665395736694, -0.9129557013511658, 0.6321684718132019, 0.8575639128684998, -0.8938085436820984, 0.39333662390708923, -0.9802290201187134, -0.13743065297603607, -1.3707793951034546, -0.4152974486351013, -1.7717620134353638, 0.6741936802864075, 1.2790335416793823, 0.834955632686615, -1.3599187135696411, -0.9479265809059143, 0.3250482380390167, -1.28084135055542, -0.9652577042579651, -0.9080570340156555, -0.651589572429657, 0.8062693476676941, 0.773975133895874, -0.3687427341938019, 1.7786905765533447, 0.6609163880348206, -0.33192178606987, -0.7342920899391174, -1.4933116436004639], [1.4654463529586792, 0.18367524445056915, 0.8454532027244568, 2.692744255065918, 1.69643235206604, -0.4285966753959656, 1.8963803052902222, -0.8167949318885803, 1.894504427909851, 0.00656706839799881, 1.2212823629379272, -1.0064138174057007, -0.4883613586425781, -0.3419985771179199, 0.17961695790290833, 0.672812283039093, -0.8844308853149414, 0.2541934847831726, -0.9553385376930237, -1.133553147315979, -0.918117880821228, -0.6863341331481934, -0.9465499520301819, 0.49146726727485657, 2.1672704219818115, 0.36623191833496094, -2.206770896911621, -0.17009437084197998, -0.5782415270805359, -0.22912965714931488, 0.2528241276741028, 1.4947127103805542, 2.2256290912628174, 0.1219683289527893, 0.34304317831993103, 0.6220210790634155, -2.387253761291504, -0.6350299715995789, 0.9450433850288391, -0.3573034703731537, -0.7809363007545471, -0.13732001185417175, -0.5331906676292419, -0.447179913520813, -2.065229654312134, -0.9879633784294128, 1.029809594154358, -1.4723832607269287, -0.08167935162782669, -1.0394971370697021, -0.7856453061103821, 0.25957995653152466, 0.7472455501556396, 0.5257085561752319, 0.5490821003913879, -0.5101768970489502, 0.9809291362762451, 0.6125028729438782, 0.06408052891492844, 0.9711549878120422, -0.36528968811035156, 0.46116751432418823, -0.085933156311512, -1.4882011413574219, 0.7986642718315125, 1.1550800800323486, 0.04486178234219551, 0.355960488319397, -0.2688848078250885, 2.0447959899902344, -1.1426640748977661, -1.5772212743759155, 0.3224051296710968, 0.5033411383628845, 0.4650344252586365, 0.530755877494812, -1.8971534967422485, 0.21797938644886017, 1.0561870336532593, -0.5933972597122192, -0.15789996087551117, -0.285077840089798, -0.684622049331665, -1.762044072151184, -0.07674819231033325, 0.3826010823249817, 2.528510808944702, 0.95737624168396, -0.4038330316543579, 0.8155797719955444, 0.9876224398612976, -1.481059193611145, -0.47185370326042175, -0.1894838511943817, 1.0252394676208496, -2.3346595764160156, 1.7707273960113525, 0.6100795865058899, -0.4055287539958954, 0.4271656572818756, 0.5324392318725586, 1.5585534572601318, 0.1042928472161293, 0.5593857169151306, -1.3852373361587524, 0.3681504726409912, -0.740074872970581, 0.286833792924881, -0.5528731942176819, 0.9198652505874634, 0.30720582604408264, -0.19045883417129517, 1.0665695667266846, -1.1712480783462524, 0.2258911430835724, -0.9796176552772522, 0.9155861139297485, -0.7171744108200073, -0.8691605925559998, 0.3268047869205475, -1.3392236232757568, -0.7591859102249146, -0.09915575385093689, -0.5963033437728882, -0.10574761033058167, 0.23859049379825592, -1.5766488313674927, -0.6275414824485779], [-1.3353960514068604, -2.9953420162200928, -0.752410888671875, -0.0819159597158432, 0.07523718476295471, 0.4901566803455353, 0.5501279830932617, 0.31843259930610657, 0.4041394591331482, 2.1168978214263916, 0.006760165095329285, 1.2318638563156128, 0.24338765442371368, -0.2001960277557373, -0.9790597558021545, 0.037579890340566635, 1.8888429403305054, 0.35836851596832275, -1.5198793411254883, 1.4385184049606323, -0.2525641620159149, -0.5768795609474182, 0.7443480491638184, -1.46532142162323, -2.0010523796081543, -0.3041386008262634, -0.11115840822458267, 0.964914083480835, 0.0032116721849888563, -0.025430472567677498, -1.069064974784851, 1.8532435894012451, -0.9465927481651306, -0.7568612098693848, 1.7082171440124512, -0.7278295755386353, 0.7506469488143921, 0.11896952986717224, -0.33989009261131287, 0.41469499468803406, 0.6610864996910095, -0.9810652136802673, -0.0029577072709798813, 0.98923659324646, 2.1963634490966797, -0.1542646437883377, 0.8417974710464478, -0.09170790761709213, 1.4681848287582397, 0.8846219778060913, 0.6617227792739868, -0.8361130356788635, 0.4143471121788025, -0.8546179533004761, 0.3102075159549713, -1.3162455558776855, -0.6531413197517395, -0.2689206302165985, -0.9466173052787781, 1.207730770111084, 0.42087382078170776, 0.09948636591434479, -0.7906738519668579, 0.8303229212760925, 0.6569917798042297, -0.3574390411376953, -0.2888610363006592, 0.5108484625816345, -0.4255475401878357, -1.7094727754592896, 1.8202592134475708, -0.7346004247665405, 0.27524274587631226, -0.3629821240901947, 0.19616153836250305, -0.939215898513794, 0.0565144382417202, -2.1108741760253906, -2.743154525756836, -1.5262773036956787, 0.27592265605926514, -0.21024636924266815, 1.0114870071411133, -0.4983111023902893, -0.5454820394515991, -0.8924893140792847, 1.8766202926635742, 0.38884714245796204, 0.07055965065956116, -0.8702963590621948, 1.5767033100128174, -0.061063386499881744, 0.28146377205848694, -0.5529982447624207, -0.44201165437698364, 0.5689774751663208, 0.1716163158416748, -0.9487428069114685, -0.6204344034194946, 0.5324710011482239, 1.5306649208068848, 0.002167449565604329, 0.2944130599498749, 0.30758923292160034, 0.4180241525173187, 0.34772709012031555, 0.9384362101554871, -2.4354593753814697, 0.32328447699546814, -0.7376670837402344, -0.9894570112228394, -0.3836098611354828, -0.7627635598182678, -0.24811096489429474, -0.33239057660102844, -0.8255242109298706, 0.9021656513214111, 0.46810874342918396, 0.7118749022483826, 2.511974573135376, -2.1563467979431152, 0.04314689710736275, 1.325605869293213, 0.533221960067749, 1.2628639936447144, 1.3798589706420898, 0.3680863678455353, 0.5794890522956848], [0.2405954748392105, -1.5118497610092163, 0.325756698846817, -0.6411396265029907, -0.10878387838602066, -1.6862951517105103, 0.0105389803647995, -0.1775815337896347, 0.609831690788269, -2.4360618591308594, 1.3537592887878418, -0.3525693118572235, 0.03195798769593239, 0.15705552697181702, 0.7501788139343262, -0.6640225052833557, 1.6560847759246826, 2.968665361404419, -0.6970561742782593, -0.645016610622406, 0.8968585133552551, -0.684639573097229, -0.7738004326820374, -1.5783660411834717, 0.1425619125366211, -1.0230108499526978, -0.5610824823379517, -0.5795208811759949, 0.5432005524635315, 2.01657772064209, 0.154083251953125, -0.5456662774085999, 0.4899080693721771, -0.6834473609924316, 0.4596123993396759, -0.6987388730049133, 2.3748607635498047, -2.0868797302246094, 0.3702852129936218, 0.3620935082435608, 0.2951459288597107, -1.6777548789978027, -0.9965100884437561, -0.8253192901611328, -0.15012574195861816, 0.6689980626106262, -1.1369463205337524, -1.324905514717102, 1.876522421836853, 0.10656296461820602, -0.3187013566493988, 0.17787747085094452, 0.4116925001144409, 1.0942537784576416, 0.41774922609329224, 0.4567722678184509, 0.8565411567687988, -0.22038163244724274, 1.3259940147399902, -0.10706070810556412, -0.41475576162338257, 0.9516972899436951, -1.0755054950714111, -0.6101139187812805, 0.13174740970134735, -0.8949400782585144, -0.8975166082382202, -0.20642732083797455, -0.8277410268783569, 0.09797857701778412, -0.2024964988231659, -0.43800801038742065, -0.7481237649917603, -0.8692882657051086, 0.6710245013237, 2.341571807861328, -0.16174612939357758, 0.564007580280304, -0.5786256194114685, -0.724014163017273, -0.02925647422671318, 0.8982763290405273, 1.4901623725891113, -1.5668164491653442, -0.16988743841648102, 0.5762169361114502, -0.07742789387702942, -0.3744259774684906, 1.275352954864502, -0.09338173270225525, 0.4521394968032837, -0.10644448548555374, -0.6073655486106873, 0.46388566493988037, 0.33463549613952637, -1.0587915182113647, 1.0036908388137817, -0.5835511684417725, 1.955021858215332, -0.2235252410173416, 0.968438982963562, 0.46856334805488586, 0.059542205184698105, 0.6568491458892822, -0.8561244010925293, 1.496412754058838, 1.5806496143341064, -2.353105306625366, -2.827162265777588, 1.025349497795105, 0.7108051776885986, -0.049762677401304245, 2.2177534103393555, -1.6787431240081787, -0.05030355975031853, -1.7309929132461548, 2.184894561767578, -1.9461942911148071, -0.8448305130004883, 0.5806670784950256, 1.187282919883728, -1.197643756866455, -0.5133927464485168, -0.3735547959804535, 0.6530118584632874, 0.6772355437278748, -1.9866960048675537, 1.1233936548233032], [1.2601014375686646, 1.8223638534545898, 0.2562079429626465, -0.3913002014160156, -0.20323620736598969, 0.776738703250885, -1.9361907243728638, 1.4589247703552246, 0.8147551417350769, 0.6487510204315186, 1.1064056158065796, -1.2532484531402588, -0.04064689576625824, -0.33987727761268616, -0.8474823236465454, -1.1528809070587158, 0.6809943318367004, 0.7126211524009705, -0.15538668632507324, -0.5600141286849976, 0.19111983478069305, 0.676646888256073, 0.08239098638296127, 0.18178944289684296, -0.724176824092865, -1.4165847301483154, 1.0906658172607422, 1.8643234968185425, 0.49306100606918335, -1.2126277685165405, -0.12988600134849548, 0.46215641498565674, -0.1398874968290329, -0.9523211717605591, -1.018816590309143, -0.22794583439826965, -0.17369060218334198, 0.08452930301427841, 0.8311104774475098, 0.6768745183944702, -1.572495460510254, 1.7346739768981934, 1.82180917263031, -0.12408078461885452, -1.0052731037139893, 0.7862291932106018, -2.229994297027588, -0.06793899089097977, 1.628556489944458, 0.8149476647377014, 0.1921597123146057, 0.5594764947891235, 1.8144402503967285, -0.9541102647781372, 1.2850528955459595, -1.419007420539856, -0.28673216700553894, 0.285288542509079, 0.12314010411500931, 1.6574727296829224, -1.0228745937347412, 0.02359776198863983, 1.129551649093628, 1.165181279182434, 1.765504002571106, -0.6931970119476318, 1.2034683227539062, -0.1141085997223854, -1.9150561094284058, -1.6384563446044922, 0.4004691243171692, 0.19281291961669922, -0.6793351173400879, 0.805971622467041, 0.1429070085287094, -0.6267737746238708, 0.48196345567703247, -1.4448561668395996, 1.0832126140594482, -1.2013691663742065, 0.34685468673706055, 0.9057902693748474, 1.4961774349212646, -0.6613532304763794, 0.4367489218711853, -1.3465543985366821, 0.759250819683075, 0.8888595104217529, 1.5120595693588257, 0.38974347710609436, -1.2724508047103882, 0.42324623465538025, -0.17717064917087555, 0.11651524156332016, 0.10605309158563614, -0.6125962138175964, 0.26377347111701965, -1.486954689025879, -0.7471980452537537, 1.5599009990692139, 1.3213746547698975, -0.5411259531974792, 0.5058639049530029, -0.058367159217596054, 2.8709118366241455, 0.3255251348018646, 1.313984990119934, 1.84244704246521, 0.1084584966301918, -1.6711665391921997, 0.342079222202301, 0.4887911379337311, 0.11322234570980072, -0.7856013178825378, -1.3769004344940186, 0.9114867448806763, -0.3411879539489746, 0.964645266532898, -0.6912674307823181, 0.8222795128822327, -1.4368150234222412, -0.2122395783662796, -0.611889660358429, -0.3051645755767822, -1.1591752767562866, 0.38218507170677185, 0.01795314811170101, 0.7895880937576294], [-1.6072345972061157, 0.5194364190101624, 0.5537908673286438, 0.8029424548149109, 1.2779549360275269, 0.1400994509458542, -0.683135986328125, -0.9759028553962708, 0.46295905113220215, 0.9401808381080627, -0.2178429663181305, 0.5071051716804504, 1.3095303773880005, -0.44828298687934875, -0.89244544506073, 0.3095513880252838, 1.5506740808486938, 0.3326164186000824, 0.5468958020210266, 0.7823431491851807, -1.1548070907592773, 1.376136064529419, -0.7178117632865906, 0.18904675543308258, 0.3140007257461548, 0.7998068928718567, 0.3227768838405609, -0.0007410713587887585, -0.01760825328528881, -1.5842700004577637, -1.6971246004104614, 0.5018733143806458, -0.2441239356994629, 0.19981920719146729, -0.8962860703468323, -0.20050637423992157, 1.3717247247695923, -1.4216196537017822, 0.19110317528247833, 2.2044451236724854, -0.6459079384803772, 0.6141767501831055, 1.357056736946106, -1.5717624425888062, -0.8437735438346863, 0.4911024868488312, 1.0125254392623901, -0.0941896066069603, -1.688604474067688, 1.0138486623764038, 0.16288435459136963, -0.7720513343811035, -1.1141985654830933, -0.2581906318664551, 0.19102834165096283, 0.13718536496162415, 0.421774297952652, 0.7774767279624939, -0.35685983300209045, 0.9950220584869385, 0.9342178702354431, -1.471872091293335, -0.12634143233299255, 0.38104474544525146, 0.010019156150519848, 1.9664301872253418, -1.0671632289886475, 1.0655972957611084, 0.46953076124191284, 0.010890057310461998, 0.4989090859889984, -0.6479557156562805, -0.6327989101409912, 2.9272189140319824, 1.6846016645431519, 1.0241422653198242, -0.8006935119628906, -0.26242774724960327, 0.6417711973190308, 0.03144882619380951, 1.5280739068984985, 0.052839457988739014, 1.208559274673462, -0.37223461270332336, -0.29684433341026306, 0.5447730422019958, -1.3638007640838623, -0.7790406942367554, 0.1923093944787979, 0.5526917576789856, -1.4234334230422974, -0.14175449311733246, 0.181832417845726, 1.5534718036651611, -2.212695360183716, -0.6373133659362793, -1.069146752357483, -0.9346048831939697, 0.4838557541370392, -1.0837326049804688, 0.5817934274673462, -1.2727290391921997, -0.05257035046815872, 0.2171773910522461, -0.0844622477889061, -0.1553105115890503, 1.2619848251342773, -0.0829009860754013, 0.7850319147109985, 0.4691321849822998, 1.5092178583145142, -0.3184289038181305, 0.2541618049144745, -1.691448450088501, -1.4176181554794312, 1.6593139171600342, 0.734061598777771, 1.1367460489273071, 1.7614470720291138, 0.10772277414798737, -0.8937868475914001, -0.05857250466942787, 1.2085503339767456, -0.36445409059524536, -1.1649843454360962, 3.074242115020752, -0.44405391812324524, -0.8293977975845337], [0.8843516111373901, -2.1013479232788086, -0.2879932224750519, 0.5599029660224915, -1.6337608098983765, -0.9368487596511841, 1.5702441930770874, 0.49524179100990295, 0.9806049466133118, 0.7161359190940857, 0.3294510841369629, -0.5811880826950073, -1.594236969947815, -1.382390022277832, -0.20061127841472626, 1.334457278251648, 0.6783052086830139, -0.11445100605487823, -0.6002176403999329, 1.3035310506820679, -0.2606273293495178, -0.7540002465248108, 1.0963199138641357, 1.4770439863204956, -0.22784683108329773, 0.8504202961921692, 0.6627345681190491, -0.517061710357666, -0.13792501389980316, -0.8487701416015625, 0.30604252219200134, 0.058706142008304596, -0.4714978039264679, 0.0344148725271225, -0.21799053251743317, 0.36506715416908264, 0.3043551743030548, -0.11030465364456177, -0.7550432085990906, -0.31558525562286377, -0.077785424888134, 0.04611353576183319, 0.8436932563781738, -0.0906948372721672, 0.06449040025472641, 0.9815415740013123, -0.6223947405815125, -0.9874258041381836, -1.0121262073516846, -0.1288614571094513, 1.2288434505462646, 0.3170555830001831, -0.23856452107429504, -1.1529910564422607, 0.9272667765617371, -1.0922961235046387, -1.540872573852539, 0.015635397285223007, 0.6754646301269531, 0.996623694896698, 1.994925618171692, 0.14809119701385498, 0.14463357627391815, 1.2346984148025513, 1.8172353506088257, -1.3649486303329468, 1.0236759185791016, -0.6807161569595337, 0.8599303364753723, -0.4301096796989441, -0.6050887107849121, 0.6594591736793518, -0.678449273109436, 0.8059970140457153, -0.15292169153690338, -0.4146842360496521, 2.143470287322998, -0.5398251414299011, 0.26549285650253296, 0.9214686751365662, -0.4561522603034973, 0.6259365677833557, -1.0525317192077637, -1.4979139566421509, 1.252479910850525, 0.35200950503349304, -1.63929283618927, 0.2079191952943802, 0.35166990756988525, -0.9098017811775208, 0.45787179470062256, -0.9391568303108215, 0.5440935492515564, 0.8419867157936096, 0.5464408993721008, 1.2363758087158203, 0.7171921133995056, -1.8898717164993286, -0.24350957572460175, 0.534080445766449, 0.894750714302063, 1.5448821783065796, -0.3245275318622589, -1.2992274761199951, 0.17356158792972565, 0.365898460149765, 0.22123682498931885, 0.708713173866272, 1.2595173120498657, 0.8734346628189087, 0.028176987543702126, 0.3324447572231293, -1.3613520860671997, 0.05688006430864334, -0.5586256980895996, -0.04733526334166527, -0.7238932847976685, 1.8550379276275635, 0.47784602642059326, 0.8814315795898438, 0.342276930809021, -1.5845412015914917, 0.2215813845396042, -1.3617655038833618, 1.0378937721252441, 1.5778379440307617, 1.001253366470337, 1.0599048137664795], [-0.40004050731658936, -0.863229513168335, 0.39488527178764343, -0.24166662991046906, 0.761246383190155, -1.1028085947036743, -0.5644538998603821, 1.234241008758545, 0.926334798336029, 2.3809468746185303, 0.9145791530609131, 0.7330376505851746, -1.5776656866073608, -2.2798359394073486, -0.5753487944602966, 0.27334555983543396, -0.3870674967765808, 0.9882023334503174, 0.7665499448776245, -0.20255263149738312, 0.9844247102737427, -0.016052741557359695, 1.1725380420684814, 2.3426475524902344, -1.121146559715271, -1.2348675727844238, 0.6734921932220459, 1.2670308351516724, -1.0026674270629883, 0.6582629084587097, -2.3244404792785645, -0.25422418117523193, 0.36722368001937866, -0.996594250202179, 2.1050963401794434, -1.2889018058776855, 0.2405441254377365, 1.703156590461731, -0.20777647197246552, 0.08536543697118759, 0.39448273181915283, 0.8165550231933594, 1.754411220550537, 0.07916560024023056, 0.7605615258216858, -0.7188020944595337, 0.14472763240337372, -1.4755895137786865, 1.9322541952133179, 0.0815141424536705, 0.31476762890815735, 0.287673681974411, -1.0828356742858887, 0.3762109875679016, -0.23979608714580536, 0.1979803591966629, 0.1580316126346588, 1.2768453359603882, 0.43344366550445557, 0.7600795030593872, -0.6936063170433044, 0.07242348790168762, -1.1496485471725464, -0.37567779421806335, 0.7899165749549866, 0.6661505103111267, 1.3910328149795532, 1.2685257196426392, 0.3003539443016052, -0.0645843893289566, -0.28641584515571594, -0.25580835342407227, 0.16384634375572205, 0.5123049020767212, 0.6911295652389526, 0.17132288217544556, 0.8093776106834412, 0.38928768038749695, 1.1139723062515259, 0.45129549503326416, -0.034898124635219574, 0.9148503541946411, -0.07190041989088058, 0.4305773377418518, 0.9349765181541443, 0.3690364360809326, -1.9559454917907715, -1.5109773874282837, 1.2501561641693115, -0.08094135671854019, -0.325648695230484, -0.4500417113304138, 1.9858030080795288, -1.3953999280929565, 0.31356504559516907, -1.193109154701233, 0.406823992729187, 0.4217297434806824, -0.1734754592180252, 1.065649151802063, 0.24116966128349304, 1.6431571245193481, 0.9437465667724609, -0.9143840074539185, 1.2504119873046875, 0.3707866966724396, -0.6578020453453064, 1.139478325843811, 0.6699252724647522, -0.12566444277763367, -0.30914250016212463, 0.2582826614379883, -0.6803615093231201, -0.3965763747692108, -0.5237388610839844, 0.7511335611343384, 0.4611302614212036, 0.8659946322441101, 1.798223853111267, -1.261795163154602, -0.00011484757851576433, 0.9644383192062378, -0.3043387234210968, -0.9906808137893677, -1.5164496898651123, -0.163413405418396, 0.13487347960472107, -0.5044279098510742], [-1.8153079748153687, 0.8078253269195557, 0.5042873620986938, -0.18223311007022858, 0.020247608423233032, 0.871911883354187, -0.3513241410255432, -0.8225369453430176, -0.7848084568977356, 0.4610625207424164, -0.5424785017967224, -1.9676837921142578, 1.4580405950546265, -0.8914045095443726, 0.7405368685722351, 0.9315861463546753, -1.5186885595321655, 0.7883533239364624, 0.15325354039669037, -0.7948853373527527, -0.7430351376533508, 1.6224263906478882, 0.5352520942687988, -0.2541077733039856, -0.23349271714687347, 1.7094494104385376, 0.31942394375801086, 0.39200636744499207, -0.44271528720855713, -1.2294706106185913, 0.28185251355171204, 0.3730887472629547, 1.448284387588501, -0.7055396437644958, -0.27088218927383423, 0.2488177865743637, 0.18410912156105042, 0.7622668743133545, -1.0913310050964355, 0.6586760878562927, -0.4811401963233948, 0.1368955820798874, 0.26518145203590393, 1.9503062963485718, 0.5960400700569153, -0.44315072894096375, -1.2007495164871216, -1.4966899156570435, 2.1061575412750244, 0.6286896467208862, 1.435515284538269, -0.9331986308097839, -0.3919677734375, 0.334842711687088, 1.6233441829681396, -0.4845096468925476, 1.9705249071121216, 1.0712677240371704, 2.519645929336548, 0.7990158200263977, -0.7030502557754517, -0.1566765457391739, -0.21105556190013885, 0.21314996480941772, -0.28870689868927, -0.7750096917152405, 0.9538158178329468, 1.731351375579834, 0.8320430517196655, 0.39059221744537354, -1.736331820487976, -0.21939148008823395, -0.3801901638507843, 0.22024866938591003, 1.8887757062911987, -1.4509031772613525, 0.6460962891578674, -2.0352518558502197, 0.2227860689163208, 0.46637749671936035, -1.143296480178833, -1.1826304197311401, -0.2191655933856964, 0.972396731376648, -2.776181697845459, 0.5934252142906189, 0.3186947703361511, 0.4020848870277405, 0.9401291012763977, -0.07201085984706879, 0.19159910082817078, 0.12888310849666595, -1.1774555444717407, -2.1630754470825195, -0.04546708986163139, -0.17641650140285492, 0.0893598347902298, -0.5497192144393921, -1.525498628616333, 0.8553206920623779, -0.5559133291244507, 1.7372499704360962, 0.7448265552520752, -2.033073902130127, -0.8202682733535767, -2.078206777572632, -2.3358094692230225, 1.7166342735290527, -0.7527779340744019, 1.6576392650604248, -0.9552232623100281, -1.7253926992416382, 0.2800029516220093, -0.3093585669994354, 0.8741387724876404, -0.6370039582252502, 0.07602062076330185, -1.3445934057235718, 0.04656523838639259, 0.21174581348896027, -0.6251677870750427, -1.2080297470092773, 0.029610373079776764, -1.7022426128387451, -1.2365778684616089, 0.4897584319114685, 1.1075935363769531, 0.36523863673210144], [-0.10857941955327988, -0.5720294713973999, -0.033927831798791885, 0.09771716594696045, 0.005457383114844561, 0.5045061111450195, -1.3549169301986694, 0.3274286985397339, -0.2618817389011383, 0.6757761240005493, -2.6267755031585693, 0.8997073173522949, -1.4892387390136719, -0.3290741443634033, 0.16219410300254822, -0.3183106482028961, -0.4214091897010803, -0.7407478094100952, 0.5280996561050415, -0.21518021821975708, 1.5319466590881348, -2.194950819015503, -0.10655223578214645, -1.446185827255249, -0.5226461887359619, -0.3249606490135193, 1.898162603378296, -0.3882076144218445, 0.79190593957901, -1.4497907161712646, -1.1731003522872925, 0.22683283686637878, -1.5082612037658691, -1.0986741781234741, -2.135762929916382, -1.7451900243759155, -0.012875430285930634, -0.821349561214447, 1.6775805950164795, 0.5513418316841125, 0.06586851179599762, -2.054985523223877, 0.10655619949102402, 0.15606586635112762, -0.6985884308815002, -0.39003893733024597, -1.1240581274032593, -0.3556614816188812, -0.5589010715484619, 0.6985534429550171, 2.1223554611206055, -0.6687219142913818, -0.05275646224617958, 0.6654011011123657, 0.8719230890274048, 1.9472163915634155, -0.8040953278541565, -1.2230660915374756, 0.5300139784812927, 0.015708358958363533, -1.2773305177688599, 0.17403344810009003, 0.33528655767440796, 0.44462403655052185, -1.2348345518112183, 1.619915246963501, 0.060232505202293396, -0.18816466629505157, 0.13282093405723572, -1.263784408569336, 2.9301629066467285, -0.16553305089473724, 1.1900049448013306, -0.21810919046401978, -0.5460704565048218, -0.29838740825653076, 0.7699726819992065, 0.4904254972934723, -0.6674764752388, 0.04952104017138481, -0.053390637040138245, 0.9813075065612793, -0.5326261520385742, 0.5410289764404297, -1.6626046895980835, -0.3747744858264923, 1.5958064794540405, -0.8170250654220581, -0.39909207820892334, -2.0387258529663086, -0.020504474639892578, 0.48221439123153687, 0.096771739423275, -1.5095558166503906, 0.09182199090719223, 0.2399439811706543, -1.3600777387619019, -1.5256569385528564, 1.0974334478378296, -0.09005942940711975, -1.583030343055725, 0.20727112889289856, -1.9409527778625488, -0.7422318458557129, -1.733941912651062, -1.1489168405532837, 1.920853614807129, 0.28228139877319336, -0.08132121711969376, 0.020979877561330795, 0.03755569085478783, 0.09360458701848984, 0.5390734672546387, 0.4910702109336853, -0.9492914080619812, -1.9105982780456543, 0.26699015498161316, 0.06082399934530258, 1.138668417930603, 0.6764362454414368, 1.024393916130066, -1.572090983390808, -0.6830836534500122, -0.46962401270866394, 0.5430772304534912, 0.5676925182342529, -1.3813210725784302, -0.8497657179832458], [-0.05868800729513168, 1.1091914176940918, -0.18489030003547668, 0.07922805845737457, 0.5863408446311951, -0.04712669923901558, 0.45088064670562744, -0.918777346611023, -0.7121881246566772, 0.6716106534004211, 0.2853718101978302, -1.9900108575820923, -0.29889899492263794, 0.4770342707633972, 1.3645455837249756, 0.3179137408733368, 0.21832506358623505, -1.711706519126892, -1.7723716497421265, -0.00901342649012804, 0.6174103617668152, -0.3918097913265228, 0.6474351286888123, 0.13786455988883972, 0.08038369566202164, 0.3930474519729614, 3.197639226913452, 1.954878568649292, -0.9523931741714478, 0.20379047095775604, -1.248695731163025, 0.5487001538276672, -0.7918563485145569, -0.27693212032318115, 0.8436934947967529, -0.10159535706043243, 0.1580624282360077, 1.1300201416015625, 1.1200743913650513, 0.44685500860214233, 0.6112799644470215, -0.7812577486038208, -1.2716578245162964, -1.2415518760681152, -0.7928018569946289, 0.08437050879001617, 0.47734928131103516, 1.2602835893630981, 0.9379948377609253, 0.4590969681739807, 0.5377900004386902, -0.3166789412498474, 0.9282041192054749, 0.3637009561061859, -0.40975889563560486, 0.08787326514720917, -0.03765091300010681, 0.3214057683944702, -0.13260988891124725, -1.441962718963623, -0.14693133533000946, 0.06714805215597153, 1.211294412612915, -1.6524945497512817, 1.96066415309906, -1.8287304639816284, -0.23299042880535126, -2.7112340927124023, 0.8942875266075134, 0.6700860857963562, 0.8679805397987366, 1.787911295890808, 0.054664645344018936, 0.5916575193405151, -0.7500579953193665, -0.3521776795387268, 0.23699431121349335, 0.8782930970191956, 1.1199795007705688, 0.6308325529098511, -0.14016997814178467, -1.2279404401779175, -0.871725857257843, 0.23924578726291656, 0.9626104235649109, 0.5565330386161804, -2.1599369049072266, -0.2642441391944885, -0.33876729011535645, 1.4381293058395386, 1.0215338468551636, 0.16418218612670898, -0.018866466358304024, -0.6785504221916199, 0.845726728439331, -1.234818935394287, -0.9953842759132385, -0.32422512769699097, 0.6328922510147095, 0.07698510587215424, 1.2664225101470947, -1.849002718925476, -0.9790756702423096, -1.5367748737335205, 1.4668476581573486, 1.1842284202575684, -0.15520349144935608, -0.41769737005233765, 1.1987009048461914, 0.044969066977500916, -0.0893770307302475, -1.1261446475982666, -1.4646601676940918, -0.5855401158332825, 0.7582114338874817, -0.7983170747756958, -0.028403211385011673, 0.9101261496543884, -0.4784810245037079, 0.3534706234931946, 0.07309792190790176, 1.9306204319000244, 1.5604090690612793, -1.3271195888519287, 0.5188176035881042, -0.7655264735221863, 0.012894729152321815, 0.660075306892395], [-1.73113214969635, 0.9182468056678772, -0.42763015627861023, -0.6173393130302429, -0.9205652475357056, -0.2135312706232071, -1.6030648946762085, -1.6993334293365479, -1.6666438579559326, 0.6502441763877869, -1.8252981901168823, 1.7013840675354004, 0.5191529989242554, -1.4165153503417969, 1.8593806028366089, 0.5750922560691833, 0.3376636207103729, -0.6063831448554993, 0.5051007866859436, 0.4164871275424957, -0.6540672779083252, -1.44746732711792, -0.20553486049175262, -0.07800283282995224, -0.401770681142807, -0.16402781009674072, 0.41848862171173096, 0.09484673291444778, 0.9992403984069824, -1.3405808210372925, 0.25955304503440857, 0.49977850914001465, 0.3276827335357666, 0.3002573251724243, 0.30047526955604553, -0.18430273234844208, -0.9391640424728394, -1.679551124572754, -0.4042257070541382, 1.6228039264678955, 2.4572227001190186, -0.8420253992080688, -1.801975965499878, -2.484201192855835, -0.017705164849758148, 0.5933727025985718, 0.8255995512008667, -0.057825036346912384, 1.5705279111862183, 1.3362282514572144, 1.2855477333068848, 0.7517538666725159, -1.0262762308120728, 0.7285793423652649, -0.2876608073711395, 0.2730899453163147, -1.5350360870361328, 0.22762486338615417, -1.7508783340454102, -0.16282369196414948, 0.7272085547447205, -0.36956632137298584, 0.5070778131484985, -0.11250805109739304, -0.42117998003959656, -0.3586413562297821, -0.6957423686981201, 1.5132321119308472, 0.025963008403778076, -0.6174997687339783, -0.3118928074836731, 0.11811242997646332, 0.29267555475234985, -0.5879919528961182, -1.2254149913787842, -0.19695721566677094, 0.11915203183889389, -0.4733143448829651, 0.48358845710754395, -0.5163387060165405, -1.7726625204086304, -1.2369955778121948, -0.44152429699897766, -0.8263751268386841, -1.5737732648849487, 1.2529869079589844, -1.8096545934677124, -1.5789698362350464, 1.016574501991272, -0.9318122863769531, 1.5877450704574585, -0.4444223642349243, -0.694972813129425, -0.5815635323524475, -0.48712873458862305, 0.03542783111333847, 1.161463737487793, -1.2172784805297852, -0.12329138070344925, -0.6226964592933655, -0.5031960606575012, 1.6146706342697144, 0.8262937068939209, 0.8398641347885132, 0.7056581377983093, 0.2537931501865387, -0.4511532485485077, -0.6328487396240234, 0.5647988319396973, -0.4298190176486969, 0.8099903464317322, -1.5246943235397339, 0.21832962334156036, 0.12393146753311157, -0.42097732424736023, 1.013255000114441, -0.10807555168867111, -0.6076095104217529, 0.2513545751571655, -0.07175663858652115, 0.49729543924331665, 0.5512686371803284, 0.2857457995414734, -0.5099329352378845, 0.8807488679885864, -0.6052001714706421, -0.3656797409057617, 0.1403171867132187]], [[0.7092687487602234, -0.20320168137550354, -0.13863083720207214, -0.37546804547309875, -1.249456524848938, -0.6568861603736877, 1.0954967737197876, -0.5365690588951111, 0.24087096750736237, -0.25321006774902344, 1.0411126613616943, 0.583898663520813, 0.7184091210365295, 0.4064091444015503, 0.32098352909088135, -1.7598485946655273, 0.5914009213447571, -0.2766363322734833, 0.6142681837081909, 1.00663423538208, 0.08557688444852829, 1.0208574533462524, 0.12040821462869644, 0.14717265963554382, -0.3936643898487091, 0.7573256492614746, 0.3983360826969147, 1.9875051975250244, -0.15326568484306335, -0.9855322241783142, -2.0332961082458496, 2.151533842086792, -1.106381893157959, -0.899321436882019, -0.04115454852581024, 0.7922659516334534, 0.18548734486103058, 1.0591145753860474, -0.3674286901950836, 1.7463054656982422, -0.6341281533241272, 0.5830602645874023, -0.20996329188346863, 0.4421499967575073, -0.7769969701766968, -0.826455295085907, 1.0072044134140015, 0.028777586296200752, -0.7365496754646301, -0.6861546635627747, -1.1443592309951782, 0.25367558002471924, -0.5054070353507996, -0.40341028571128845, -0.06150161102414131, -0.9223123788833618, 2.3483035564422607, -1.6042289733886719, -0.13639815151691437, 1.3023484945297241, 0.07024402916431427, 0.8715518712997437, -2.8833134174346924, 0.1200525090098381, 2.308652877807617, 0.7305607795715332, 1.4148743152618408, -0.2131415158510208, -0.5164387822151184, -0.3882403075695038, -0.8256806135177612, -0.6599619388580322, -0.5695491433143616, -0.8268934488296509, 2.459015130996704, -0.5348793864250183, -0.2463926076889038, -0.09994359314441681, -1.858258605003357, -0.14749523997306824, 0.023122450336813927, 0.3655782639980316, 0.8459494113922119, -2.1575610637664795, -0.8332693576812744, 0.06472863256931305, 2.0015671253204346, 0.8745917677879333, -0.2121133804321289, -0.2542386054992676, -0.3003354072570801, 0.7807336449623108, 2.8435122966766357, -0.1767987757921219, -0.2622232437133789, 1.661075472831726, 0.1484222710132599, -1.5775527954101562, -0.9100218415260315, 1.313305377960205, 0.7669659852981567, -0.0805722326040268, 1.9393846988677979, 0.9587033987045288, -0.3843742311000824, 1.2427419424057007, 1.5810067653656006, -0.5318490266799927, 1.3557686805725098, -0.03794525936245918, -1.221608281135559, 0.8773949146270752, 0.37095898389816284, 2.0778207778930664, 0.1705092191696167, 0.03216032311320305, 2.126124143600464, -1.2635846138000488, 0.6413781046867371, 0.7216143012046814, -0.14232677221298218, -1.6269237995147705, -0.43613147735595703, 0.301547110080719, -1.755039930343628, -1.096153974533081, 0.8912754058837891, 0.40076354146003723], [0.400269091129303, 0.26610684394836426, -1.1648823022842407, -2.348397970199585, 1.7886582612991333, 0.4784895181655884, -1.308014154434204, 1.109972357749939, 0.5090744495391846, 1.159896969795227, 1.766268014907837, 0.5009715557098389, -2.2467315196990967, -0.67254239320755, -1.3214499950408936, -1.0509264469146729, -0.6697022318840027, -0.09912795573472977, 1.0333831310272217, 1.4180799722671509, 0.6320785880088806, 0.19267535209655762, 0.8558760285377502, 0.1195460706949234, 1.0980212688446045, -0.5019252896308899, 0.05488855764269829, -0.3101898431777954, -0.10588382929563522, 0.9034163355827332, 2.834239959716797, -0.629165530204773, -0.9083101749420166, -0.21641720831394196, -1.158443570137024, 0.1405792236328125, -0.718981146812439, -0.6357595920562744, -0.27795785665512085, -0.9939441084861755, -0.12232191115617752, 0.32127732038497925, -0.24037636816501617, -0.6110310554504395, -0.11092615127563477, 0.5969594120979309, 0.591119110584259, -0.7346266508102417, -0.6995800733566284, -0.5479962825775146, -0.22476403415203094, -0.08080621063709259, -0.11418921500444412, -0.3500964343547821, 0.4716379940509796, 0.1864464282989502, 0.4351588785648346, 0.46544933319091797, 0.9956181049346924, 0.08393270522356033, -1.3845049142837524, -1.104371190071106, -1.2717221975326538, -0.16020843386650085, 0.843513011932373, 0.8392074704170227, -1.7605501413345337, -0.2641998529434204, 0.8878825902938843, -1.0401862859725952, -0.40634557604789734, 1.566533088684082, -0.4609215259552002, -0.6716131567955017, -1.55593740940094, -2.2213947772979736, -1.3831977844238281, -0.7481151223182678, -1.4531047344207764, -1.5395718812942505, 0.23886363208293915, 0.7916027307510376, 1.3894636631011963, -0.8463796973228455, 0.05532638728618622, -1.015317440032959, 0.23495739698410034, 0.9587277770042419, 1.132332444190979, -1.906647801399231, -0.08552248030900955, -1.4656720161437988, -0.3575047552585602, 0.7656615376472473, -0.6089736819267273, 1.2185055017471313, -0.11090577393770218, -0.2981577217578888, 1.0017253160476685, 1.4690732955932617, -0.6537963151931763, -1.5031554698944092, -0.8573671579360962, 0.3333317041397095, -1.4121991395950317, -0.19817940890789032, -0.09791368991136551, -0.5106222033500671, -1.8337395191192627, -0.06381623446941376, -0.21107204258441925, 0.31669673323631287, -0.20443519949913025, 1.4293590784072876, -1.2973588705062866, -0.4766887128353119, -0.9099008440971375, 1.0954904556274414, -1.3791996240615845, -0.2891206741333008, 0.8936399817466736, 0.47740665078163147, 1.4088809490203857, 0.08734960108995438, 0.3269284665584564, -0.25886431336402893, 0.05775168165564537, 0.3192734122276306], [-0.03075260855257511, -0.42660441994667053, -1.5313642024993896, -0.43351882696151733, 1.3349361419677734, 1.8064254522323608, -0.7864043116569519, 0.3760395348072052, 0.9474992156028748, 2.1414270401000977, 0.3374793529510498, 3.001293897628784, 0.06972507387399673, -0.358016699552536, -0.9813661575317383, 0.382296621799469, 1.2562837600708008, 0.202593594789505, 0.5228489637374878, -2.539794921875, -0.8345608711242676, 0.4588116705417633, -0.35683131217956543, 0.08301027864217758, -0.14668558537960052, 0.0525248683989048, 0.7597121000289917, -1.2109582424163818, -0.7776340842247009, -0.290404736995697, 1.7846553325653076, -1.6162723302841187, -0.5106021761894226, 0.9680299758911133, 0.6212196946144104, 0.7463685274124146, 0.42683473229408264, 0.18067730963230133, -0.09401005506515503, 0.6303634643554688, -0.26279473304748535, 0.7867183089256287, -1.2819702625274658, 2.0853097438812256, 1.4893519878387451, 0.5429922938346863, 0.36617377400398254, 0.012631328776478767, 0.4731198847293854, 1.396546483039856, -0.6754907965660095, 1.1165047883987427, 0.4840319752693176, 0.7560634613037109, 1.0880428552627563, -1.1110526323318481, 1.5659948587417603, 0.07575690001249313, -0.9521670341491699, 0.5964251756668091, 0.10509762167930603, -1.9197123050689697, 0.2340039759874344, 1.6042168140411377, 0.06078249216079712, 0.09824126958847046, 1.291195034980774, 1.1901146173477173, 0.30190950632095337, -0.6137492656707764, -0.010142873041331768, 0.5239965915679932, 0.018358150497078896, -0.6955366134643555, 0.6749096512794495, 0.9630789756774902, 1.11224365234375, -1.330153465270996, 0.013473420403897762, 0.1817253828048706, -0.7544295191764832, 1.3363087177276611, 0.08779285848140717, 0.9525443315505981, 0.6533162593841553, -0.2606690526008606, -1.5144973993301392, -0.48713207244873047, 1.1075152158737183, -1.6562206745147705, -1.0106685161590576, -0.40076810121536255, 0.7896002531051636, -1.6429136991500854, -0.3504306674003601, 0.0396369993686676, 1.1736470460891724, 0.021533675491809845, 0.8272342681884766, -0.08490483462810516, 0.6838119626045227, 0.6513735055923462, 1.0174793004989624, 0.2050531804561615, 0.13132093846797943, -1.0922361612319946, 2.707157611846924, -2.598557710647583, -0.22901524603366852, 0.9571366310119629, -0.5466598272323608, 1.0493285655975342, -0.7754207849502563, 2.2310683727264404, -0.0317552424967289, 0.7937690019607544, 0.9440647959709167, -0.9709541201591492, 1.9538761377334595, 0.20119015872478485, 0.12062931060791016, 0.8080053329467773, 1.2307389974594116, 1.1127746105194092, -0.003142997156828642, -1.1604524850845337, -0.28426113724708557, 0.08360755443572998], [-0.3870474696159363, -0.7878292798995972, 0.7461104989051819, 0.9986521601676941, 0.5663663148880005, 0.24713489413261414, -1.966611623764038, -0.9449243545532227, 1.0069867372512817, -0.3000069260597229, -1.9886914491653442, -1.3022047281265259, -0.7811026573181152, 1.0714236497879028, -0.19190505146980286, -1.3288650512695312, -1.6593079566955566, -1.5584322214126587, 0.14676225185394287, 0.1596257984638214, -1.6658680438995361, -0.26013725996017456, 2.3542850017547607, -0.7179425358772278, 0.3717830181121826, -1.671447992324829, 1.3835150003433228, 0.18178577721118927, -1.260719656944275, 0.6004257202148438, 0.41170087456703186, -0.5852554440498352, -0.4381764829158783, 0.23941117525100708, 1.6283844709396362, 1.2286744117736816, 0.12280172854661942, 1.8919461965560913, -0.7558916807174683, -1.2015448808670044, -0.2209738940000534, -1.1277797222137451, -1.013577938079834, -0.4515555202960968, -0.35828283429145813, -0.12205829471349716, 0.06883306056261063, -0.18514224886894226, 1.4317095279693604, 0.6957716345787048, 0.7380773425102234, -0.6251410841941833, 0.5483580827713013, -0.8741816282272339, 0.19606247544288635, 0.1492414027452469, 0.3920796513557434, 0.05475408211350441, 0.6804198622703552, 0.30470776557922363, -0.1579870879650116, 1.057775855064392, 1.9278823137283325, 0.5648207068443298, -0.7681277394294739, -0.9939410090446472, 0.05810623615980148, -0.06100653111934662, -0.2501624822616577, 1.5758137702941895, 2.190275192260742, 1.934922218322754, 0.19777975976467133, 0.32109495997428894, 0.051712363958358765, -0.295759916305542, -0.09177219867706299, -1.6428788900375366, -1.6755117177963257, -0.8145273923873901, -0.7787109613418579, 0.7778132557868958, 0.8193528056144714, 0.1544055938720703, -0.2956354320049286, 0.4847337007522583, 0.33263587951660156, -0.38527849316596985, -0.9336233735084534, -0.6606960892677307, -0.7142543196678162, -0.8961703777313232, 1.1658579111099243, -0.4184688627719879, 2.1233391761779785, -0.28452393412590027, -0.9285553693771362, 0.3461253345012665, 0.5727793574333191, 0.2182760238647461, -0.5403599739074707, 1.2927809953689575, -1.9292657375335693, 0.8124836087226868, 1.461207628250122, -0.3893721103668213, -0.0026895685587078333, -1.3704947233200073, 0.7667962312698364, -0.4757712185382843, -0.18225541710853577, 0.21708862483501434, -0.4392130672931671, -1.1332530975341797, 0.5216628313064575, 0.991195023059845, -1.6351102590560913, -1.3447184562683105, 0.5096409916877747, -1.2118690013885498, -0.31537383794784546, 1.7197288274765015, 0.9242923855781555, 0.16213776171207428, -1.872148036956787, 0.6147135496139526, -1.4332894086837769, -0.7211500406265259], [1.026759386062622, -0.1542343646287918, -1.0383708477020264, -1.0129331350326538, 0.8224045038223267, -1.5319383144378662, 1.5111364126205444, 1.2019723653793335, -0.6865673065185547, -0.1266840100288391, 0.3413327634334564, 1.7391825914382935, -0.3683096170425415, 0.004927521571516991, -0.4506724178791046, 1.3095545768737793, -1.3489147424697876, 1.877783179283142, -0.24204543232917786, 0.7393370866775513, 1.1493533849716187, -0.61065274477005, 0.05557619780302048, 1.7610784769058228, 0.8288048505783081, -0.3531903922557831, 0.7410503029823303, 0.4871661961078644, 1.3419231176376343, -1.0709997415542603, -0.5659618973731995, 0.8393444418907166, 0.46768832206726074, -0.4012333154678345, 0.13222381472587585, -0.7644478678703308, 0.09687431901693344, -0.4642292261123657, 1.8192250728607178, -0.19623146951198578, 1.7257128953933716, 0.9692953824996948, -1.5503714084625244, 0.439892053604126, 1.3921115398406982, 0.4590563476085663, -0.6752352714538574, -0.4881178140640259, -0.5992194414138794, -1.5781790018081665, 0.5284174084663391, -0.17479613423347473, -0.7536983489990234, -1.5324422121047974, -0.4341104030609131, 0.937042772769928, -2.2324178218841553, 2.506884813308716, -2.7073934078216553, 0.08998233079910278, -2.047315835952759, -0.486186683177948, -0.05135622248053551, 0.6932682394981384, -0.8167274594306946, 1.4368584156036377, -0.011066419072449207, -0.613097608089447, -0.8290817737579346, 0.5289822220802307, -1.3249398469924927, -0.6994158625602722, -0.9550244808197021, -1.4335500001907349, -1.2177636623382568, 1.3833335638046265, 0.8827612996101379, 0.12282617390155792, -0.5882370471954346, -0.3664354085922241, 0.6113741993904114, -0.8585385084152222, 2.1808745861053467, -0.19161593914031982, -0.28464561700820923, 0.5017499923706055, -0.4567515254020691, -0.3829675018787384, 0.2033526599407196, -0.6441740989685059, -0.4628946781158447, -0.41496556997299194, -0.5983299016952515, 0.34171077609062195, 0.41160839796066284, 0.38408491015434265, -2.3184545040130615, -1.485312819480896, -0.8392300605773926, 1.4961862564086914, 1.195526123046875, 1.1027214527130127, 1.1489300727844238, -0.963104784488678, -0.14665301144123077, 0.6908072829246521, 2.1303763389587402, 0.7673566937446594, -0.8577252626419067, 0.3113975524902344, -0.25829774141311646, 1.328564167022705, 0.06480224430561066, -0.1735495924949646, 0.16792722046375275, 0.03389603644609451, 0.39024367928504944, 0.6623152494430542, 0.19272366166114807, -1.3766814470291138, -0.3040110170841217, -0.7742003202438354, 0.5364235639572144, 0.6944186091423035, -0.19364579021930695, 0.3655436635017395, -1.514757752418518, 1.0334059000015259], [-0.6802090406417847, -0.0381212942302227, -0.8868297934532166, -1.3608934879302979, 1.172532081604004, 1.620671033859253, -0.08695891499519348, 0.8136371374130249, 0.013885146006941795, -0.07918055355548859, 0.22181597352027893, 1.5650744438171387, -0.7625210285186768, -0.7326027154922485, -0.09041711688041687, 0.402533620595932, 1.002031683921814, -0.1444168984889984, 0.7003936767578125, 0.6914198994636536, -2.6416733264923096, 0.3994777202606201, 1.2729825973510742, -0.4130367040634155, 3.0614864826202393, -0.5893877744674683, 1.830504298210144, -1.6214044094085693, -0.9462577104568481, -0.5638047456741333, 1.7975518703460693, -0.10186747461557388, 0.3041166663169861, -1.040428876876831, 0.15389178693294525, -1.1991358995437622, -1.2584586143493652, -0.9078694581985474, 1.342302918434143, -1.973261833190918, -0.2113548219203949, 1.3081703186035156, 0.026266491040587425, -1.0689982175827026, 0.5197991132736206, -1.0364065170288086, 0.03714723885059357, -2.263758897781372, 1.0603224039077759, -0.06306527554988861, 0.7074807286262512, 0.24006740748882294, -0.2715763449668884, -0.556873619556427, 0.6410622596740723, -1.0387465953826904, -0.9774051308631897, -0.2133231908082962, -1.8108747005462646, 0.44335269927978516, 0.5032480359077454, 0.7251790761947632, 0.40443795919418335, 0.23289044201374054, 0.9436834454536438, -0.6409249305725098, -1.4134138822555542, 0.8651560544967651, -0.22458744049072266, 0.19377927482128143, 1.0526235103607178, 1.4163479804992676, -0.24170297384262085, -0.4915701150894165, -0.5268431305885315, 1.6998693943023682, -1.3224588632583618, 1.0655696392059326, -0.021789923310279846, -0.5011213421821594, -1.2508370876312256, 0.601230800151825, 0.21388418972492218, -1.3623602390289307, -0.597891628742218, -1.4317305088043213, -0.1319926232099533, -1.357466697692871, 0.525328516960144, 1.0136933326721191, -1.150900959968567, -0.8000048398971558, -0.0031734113581478596, -0.09085407853126526, -1.4766812324523926, -0.020174626260995865, 1.0061681270599365, -1.167273998260498, 1.5877797603607178, -2.029967784881592, 1.6670379638671875, 1.4944562911987305, 0.6345493197441101, -1.3315426111221313, 1.009761095046997, -1.051296353340149, -1.963562250137329, -0.9193534255027771, -0.12743601202964783, 1.4289275407791138, 0.9674599766731262, 1.006443977355957, 0.8131321668624878, 0.6298744082450867, -0.7392395734786987, 0.5440111756324768, -1.254503607749939, -1.687049388885498, 0.48386046290397644, 0.8335806727409363, 0.03256234899163246, 0.16860519349575043, 1.5592032670974731, -0.6497312188148499, -2.0956246852874756, 0.4941372871398926, -0.8396736979484558, 1.3695875406265259], [-0.8763875365257263, -0.1561623215675354, 0.3779454231262207, -0.501110315322876, 2.0249392986297607, -0.3187180459499359, 0.7352408170700073, -2.0079431533813477, 0.2866528332233429, 1.6728962659835815, -0.6702694296836853, 0.7040331363677979, 1.012486457824707, -1.6496821641921997, 0.08104439824819565, 0.787092387676239, -1.27250337600708, 0.9402190446853638, 0.48365113139152527, 0.05168088898062706, 0.3638756275177002, -1.844963788986206, -0.9354264140129089, 0.46232396364212036, -2.465555429458618, -0.2010132372379303, 0.17273053526878357, 1.507622241973877, -0.6906132102012634, -2.0557196140289307, 1.3715978860855103, 0.09042337536811829, -0.02114362083375454, 1.6805009841918945, -0.6084542870521545, -1.078203797340393, 0.592478334903717, -1.0561373233795166, 0.6478378176689148, 0.31022265553474426, 1.3848720788955688, -0.7433308362960815, 0.04366592317819595, 1.3283798694610596, -0.28955167531967163, 0.14438486099243164, 0.2977035939693451, -0.24971167743206024, -0.10585714876651764, 0.14813639223575592, 1.3451015949249268, 0.27908197045326233, -0.4575069546699524, -1.1027365922927856, 2.3911449909210205, -0.37428686022758484, 0.9276080131530762, 0.952674925327301, 1.844151258468628, 0.5318236947059631, 0.14450788497924805, 0.1423112154006958, 0.6186539530754089, 0.38950854539871216, 1.3251651525497437, -0.1722797304391861, 0.01701727695763111, 1.051041841506958, 1.784409523010254, -0.42271432280540466, 1.0403716564178467, 2.2751262187957764, 0.995087206363678, -0.37949511408805847, 0.4174516797065735, -0.5628248453140259, 0.844855010509491, -0.3093620836734772, -1.6926724910736084, 0.2980923354625702, 1.498002290725708, -0.33169081807136536, -0.767815887928009, -1.099731683731079, -0.3909953832626343, 0.025231556966900826, -0.5873327851295471, 0.47646525502204895, -0.3536296486854553, 0.1000557392835617, -0.7821999192237854, -1.4480445384979248, -0.02285962551832199, 0.23642931878566742, 1.602216362953186, 0.833286464214325, -2.450263738632202, 0.35394981503486633, -0.12048360705375671, -0.7417399287223816, 1.8607170581817627, 0.7867933511734009, 0.8223796486854553, 0.15711793303489685, -1.282301664352417, -1.696537971496582, -0.1787058562040329, 1.8137400150299072, -0.5460693836212158, 0.3825575113296509, -0.47115835547447205, 1.0896060466766357, -0.5569732189178467, -1.625491738319397, 0.9832674264907837, -0.32608485221862793, -0.6435995697975159, 2.4675137996673584, 0.5725710988044739, 0.25598573684692383, -2.0334200859069824, 0.7861640453338623, 0.5735169053077698, 0.2732727825641632, 0.1035376712679863, 1.3655345439910889, 0.3770922124385834, 0.22536151111125946], [-0.6419343948364258, 0.1674744188785553, -1.0921941995620728, -0.6061071753501892, 1.5622844696044922, -0.11512346565723419, -0.6450058221817017, 0.7494713068008423, -0.7545120716094971, -0.6999077200889587, 0.7511868476867676, -0.10479520261287689, 0.048890698701143265, -0.3420252799987793, 0.9613546133041382, -1.1891472339630127, -0.639009416103363, -0.727454423904419, -0.4754088222980499, 0.5906060338020325, -1.1795127391815186, 0.5310497879981995, -0.3026566505432129, -0.02757118083536625, 0.004160932265222073, 0.9832333326339722, 0.529897928237915, -0.25296705961227417, 0.6499790549278259, 0.06511451303958893, -1.2407056093215942, 1.0752853155136108, 0.9853940606117249, 0.2560650110244751, 0.3144514262676239, -0.7742940187454224, 0.9474967122077942, 1.2293130159378052, -0.38597622513771057, 0.8847836852073669, -0.0061131250113248825, 0.5467858910560608, -0.22384391725063324, 1.2366758584976196, -0.5188360810279846, -0.1428484469652176, 0.5264134407043457, 0.015788137912750244, 0.9668489098548889, -0.9183675050735474, 0.17754757404327393, 0.8587479591369629, 1.1562294960021973, 0.5593533515930176, -0.6633176803588867, 0.4518435001373291, 0.986039400100708, -1.721335530281067, -2.602858066558838, -0.2891199290752411, 1.323773741722107, -1.1024856567382812, 0.2519564628601074, -0.028112120926380157, -2.1934540271759033, 0.30084893107414246, -0.40818604826927185, 1.5991953611373901, 1.237851858139038, -0.06998566538095474, 0.16474764049053192, 0.15234890580177307, -0.44542139768600464, -0.28918638825416565, 0.3116535544395447, 0.9658979177474976, -0.868868350982666, 0.18143229186534882, 1.0046956539154053, 0.14440582692623138, 1.531350016593933, -1.231758713722229, -0.683907687664032, -0.5250805616378784, -0.44250279664993286, 0.5819962024688721, -0.43034252524375916, -0.08273313194513321, -0.3231901228427887, -1.4159724712371826, 1.9844549894332886, -3.302983045578003, -1.6525828838348389, 0.2791787087917328, -1.1899635791778564, -0.7869264483451843, -1.0877065658569336, 0.021869000047445297, 0.20618537068367004, 1.5326367616653442, -1.2903093099594116, -0.7796393036842346, -0.2564971446990967, 0.29657694697380066, -0.23765218257904053, 0.4469785988330841, 0.8703430891036987, 0.8042445778846741, -2.2060070037841797, 0.36584392189979553, 0.012393990531563759, 0.4398359954357147, 0.3361107409000397, 0.042776793241500854, -0.6555916666984558, 0.21907301247119904, 0.044680505990982056, 1.9927953481674194, 1.1914968490600586, -0.7001857161521912, -0.03766924515366554, -0.6359060406684875, -1.0129824876785278, 0.09710250794887543, 0.07168715447187424, -0.043135300278663635, 0.7730104327201843, -1.076249122619629], [0.6969439387321472, 2.748119831085205, -0.30374735593795776, -0.8225637674331665, 1.0499545335769653, -0.5407237410545349, 3.421833038330078, -1.8246653079986572, -1.1193368434906006, 0.7119672894477844, 0.5101988315582275, 0.6426141262054443, 1.1394925117492676, -1.552628993988037, -0.47576555609703064, -0.5323406457901001, -0.001705366070382297, -0.7444146275520325, 0.7769551873207092, 1.155241847038269, -1.2924822568893433, -0.8977147340774536, -2.55692195892334, -0.1479087471961975, 0.1539502888917923, 0.4221550226211548, 0.38079795241355896, 0.18883228302001953, -0.43356382846832275, -1.072940468788147, -0.2698734700679779, 2.4670348167419434, -0.1970262974500656, 0.7938339710235596, 0.9780066609382629, 1.937678337097168, -0.20434053242206573, -0.8609226942062378, 0.267107218503952, 0.9469776153564453, 1.078739881515503, -0.6635777354240417, -0.17198097705841064, -0.5136890411376953, -1.1756693124771118, -0.2803657650947571, -0.10650154948234558, 0.058851368725299835, 0.912635862827301, 0.9625934362411499, -1.172288417816162, 0.6135467290878296, 0.5580441951751709, -0.2385406643152237, 0.3079081177711487, -0.4961691200733185, 1.8532812595367432, 0.7473247051239014, -1.7851786613464355, -0.4483378827571869, 0.7847615480422974, 0.343625545501709, 1.3608978986740112, -0.31752970814704895, -0.17316527664661407, -0.1957050859928131, -0.2876302897930145, -0.020897073671221733, 1.415794014930725, -0.5739213228225708, 1.3226717710494995, -0.3261276185512543, -0.09757047891616821, 0.8982137441635132, -0.6053368449211121, 0.5752878189086914, -1.009193778038025, 2.5501441955566406, -0.20771493017673492, 0.07365094870328903, 1.512579083442688, -0.033326469361782074, 1.049270510673523, 0.9507396817207336, 0.36233511567115784, 0.5575308799743652, -1.0894711017608643, 0.882366418838501, 1.024701714515686, 0.3871883749961853, 0.27352333068847656, 0.3071432411670685, -1.3576561212539673, 0.058432627469301224, 0.9106401205062866, -1.1178252696990967, -0.940705418586731, 0.14496468007564545, 0.7827134728431702, 0.9986245632171631, -1.4595444202423096, -0.9069181680679321, -0.3480609059333801, -0.25411176681518555, -0.4025869071483612, -0.3557490408420563, 1.1330246925354004, 0.3732827603816986, 0.9274373054504395, 1.3206655979156494, -0.5036607384681702, 0.3021314740180969, 1.9160614013671875, 1.977389931678772, -0.46490710973739624, 0.0005682809860445559, 1.0307259559631348, 0.7368003726005554, -1.2137309312820435, -1.0493053197860718, 0.12407442182302475, 1.332228660583496, 0.07443738728761673, 0.7590712308883667, -0.0665302574634552, 0.6544529795646667, -0.7133255004882812, -0.4428422451019287], [-1.6778568029403687, 0.5539824366569519, -1.0815268754959106, 1.226054310798645, -0.21035312116146088, 0.9474044442176819, 0.07888190448284149, 1.5191560983657837, 0.22997401654720306, 1.0853818655014038, -2.1629605293273926, 0.9340372681617737, -0.9198853373527527, -1.550840973854065, 0.5249000787734985, -1.0053505897521973, -0.48026034235954285, 0.6882045865058899, -0.1648423969745636, 2.3035244941711426, 0.49308547377586365, 0.18673105537891388, 0.4884468615055084, -2.60033917427063, 0.18622057139873505, 0.20851776003837585, 0.9504717588424683, 0.33939793705940247, 0.9006252884864807, 1.0453357696533203, -1.621420979499817, 1.0260037183761597, 1.17453134059906, -1.1989877223968506, -1.8121832609176636, -0.3724548816680908, 0.2977019250392914, 0.18027862906455994, 0.4443555474281311, 0.42007768154144287, 1.509411096572876, -0.7743093967437744, -1.0042548179626465, -0.2891785502433777, 0.5404310822486877, -0.07315845042467117, 0.5559026002883911, 0.16546984016895294, 2.0723745822906494, -0.31225019693374634, 0.06327317655086517, 1.306256890296936, 0.943331778049469, -0.7666305899620056, 0.6850379705429077, -1.0113037824630737, -0.6211023926734924, -0.3821985423564911, 0.38959601521492004, 1.1503206491470337, 0.20322348177433014, -0.01544963475316763, 0.2918471097946167, 0.8700824975967407, 2.129812479019165, -0.8565937876701355, -0.912752628326416, -0.03727724403142929, -0.7350265979766846, 0.0797610804438591, 1.143908977508545, 1.7813915014266968, 1.630519986152649, 0.7398775815963745, 0.2652873396873474, -0.717513382434845, -0.8822833895683289, -0.7629284858703613, 0.21986956894397736, -1.1986186504364014, -1.5529338121414185, 1.7874473333358765, -0.4258940517902374, 0.7774989604949951, -0.6446424126625061, -1.1827484369277954, 1.1587039232254028, 0.06369363516569138, -0.5021671056747437, 1.9042658805847168, -0.10514558106660843, -0.0007432823185808957, 0.254025399684906, 0.711624026298523, -0.20610307157039642, -1.1920620203018188, 0.14288897812366486, 0.0021000662818551064, 0.6431268453598022, -0.021728256717324257, 0.28206104040145874, 0.03976478800177574, 0.43864431977272034, -1.3207060098648071, 0.9815136194229126, 0.3098333477973938, 0.04587053135037422, 1.6459083557128906, -0.8350444436073303, 0.8572689890861511, -0.3180168569087982, 1.3023033142089844, 2.572141408920288, 0.1004929170012474, 0.6058701872825623, 0.5136597752571106, 0.6332739591598511, 0.24398531019687653, -1.6774461269378662, -1.0385410785675049, -1.5518510341644287, 1.150303602218628, -0.6619758605957031, -0.07430989295244217, 1.884581446647644, 0.5961540937423706, 0.9021658897399902, -0.09970016777515411], [-1.0166349411010742, 1.9466681480407715, -0.49977225065231323, 0.8028114438056946, -0.3159472346305847, 1.3038634061813354, 1.9750839471817017, -0.33908066153526306, 0.8179413676261902, -0.2734464406967163, 0.3803466558456421, -0.6966875791549683, -0.04215306416153908, 0.525719165802002, -0.9434873461723328, -1.0249489545822144, -0.9751304388046265, -1.5289567708969116, 1.932062029838562, 0.23478427529335022, 1.489903211593628, 0.1982133835554123, 0.2937825620174408, 0.47404515743255615, -0.798965334892273, 1.2010655403137207, -0.5926843285560608, 0.5664873123168945, -0.2586497366428375, -1.2438054084777832, -1.286069631576538, -0.15323692560195923, -0.6343416571617126, -1.698049545288086, -2.5805325508117676, 2.1833810806274414, -0.13286302983760834, 0.6680908799171448, -1.2489205598831177, -1.2485216856002808, 0.2778037488460541, -0.5540953874588013, 0.5314205288887024, -0.04066364839673042, -0.009737844578921795, 0.5457767844200134, -1.5590393543243408, -0.34097230434417725, 0.28694576025009155, 0.6928521394729614, -0.7039175033569336, 2.6801185607910156, 1.3951373100280762, 1.2174097299575806, 0.32686078548431396, 1.644940733909607, 1.4673969745635986, -1.4121450185775757, -0.9337681531906128, -0.7039015889167786, 1.0391435623168945, -1.1825826168060303, 0.4134405255317688, -0.7120397686958313, 1.2842497825622559, 0.998072624206543, -2.1842920780181885, -0.6418226361274719, -0.282464861869812, -0.07282518595457077, -0.39610204100608826, -0.6481062769889832, 1.3859301805496216, 1.534595012664795, -0.5770615935325623, -0.7933427095413208, -0.10155924409627914, 0.5877915024757385, -1.1849274635314941, 0.6923356652259827, -0.48473092913627625, -1.4066786766052246, -0.4250447154045105, -0.30002644658088684, -0.48678910732269287, 0.7464351654052734, -0.24934647977352142, 0.5671626329421997, 1.8080586194992065, 0.4640628695487976, 1.5085117816925049, -0.5316402316093445, -0.3242378830909729, 0.8337096571922302, -1.5974416732788086, 0.4936513900756836, -1.1598598957061768, 0.44140052795410156, -1.8756300210952759, -1.134046196937561, 0.5452029705047607, -0.22791673243045807, -0.21890436112880707, 0.3848363161087036, -0.7685290575027466, 0.18027089536190033, -0.4306141138076782, -0.1890295445919037, -0.42882388830184937, 0.18290483951568604, 1.3527172803878784, 0.7123534083366394, 0.6396936178207397, -0.3927161693572998, -1.974341869354248, 0.8873335123062134, 0.01856599561870098, 0.02919350378215313, 2.3815810680389404, -0.43835100531578064, 0.2277344912290573, -0.5374696254730225, -1.3993477821350098, -0.6912413239479065, 0.8521578311920166, -1.6977035999298096, 1.0263925790786743, 0.0006530286045745015], [0.7391082644462585, 1.1126246452331543, 0.04059028625488281, -0.038232650607824326, 2.4023196697235107, 0.229835644364357, 1.2331572771072388, -1.6723904609680176, 1.7363941669464111, -0.7307560443878174, 1.5535749197006226, 1.0712697505950928, -0.8914204239845276, 0.023903673514723778, -0.45935848355293274, -0.3604285418987274, -0.0969395861029625, 1.462365984916687, -0.12854142487049103, -1.6157158613204956, 1.0654455423355103, -1.1640949249267578, -0.8567683696746826, 0.829622745513916, 0.6330804228782654, -0.07732705026865005, 0.7750499248504639, -0.6052854657173157, 0.959324300289154, 1.5727413892745972, -0.2594141364097595, -1.3042035102844238, 1.5083394050598145, -1.4878618717193604, 0.699971616268158, -1.1217679977416992, 1.1379350423812866, -0.4561820924282074, -1.5473411083221436, 0.8774161338806152, -0.9152786135673523, -0.1546088606119156, -0.5734562277793884, 0.4330156445503235, 0.2123880237340927, 2.6001946926116943, 0.980176568031311, -1.1248128414154053, -0.5294883251190186, -0.45901259779930115, 2.3247127532958984, -0.6631133556365967, -0.09834101051092148, -0.2536991238594055, -1.234980583190918, 0.8626635670661926, 1.897612452507019, 1.441976547241211, 1.1896240711212158, 0.7660006880760193, 1.8732938766479492, -0.07901076227426529, -0.9888986349105835, -0.990675151348114, -0.5304122567176819, 0.40051132440567017, 0.4515710771083832, 0.21452850103378296, -0.30745914578437805, -1.4308451414108276, -0.2633509039878845, -0.48529958724975586, -0.031694818288087845, -1.7530276775360107, 0.21181100606918335, 0.1863095611333847, -0.8973348140716553, 0.5362256169319153, 2.7685742378234863, 1.764744758605957, -0.6272173523902893, 0.4788566529750824, 0.6915478706359863, -1.0107691287994385, -0.7563648819923401, 1.6809512376785278, 1.0547494888305664, 0.12623564898967743, 0.13847440481185913, -0.481376975774765, 1.1369578838348389, 0.8602330088615417, -0.44062355160713196, 0.27811312675476074, 0.04435677453875542, -0.6695832014083862, 0.2787308692932129, -1.0268667936325073, -0.1870855689048767, 0.5036542415618896, 1.4589147567749023, -0.1857605278491974, 1.3492865562438965, -0.21863295137882233, 1.0543856620788574, 1.4813194274902344, 0.47946053743362427, -0.15864482522010803, -2.771566390991211, -1.717463731765747, 0.2007008194923401, -0.19369076192378998, -0.033072102814912796, 0.9733387231826782, -0.3469785749912262, 0.7027660012245178, 0.3216571807861328, -1.2951806783676147, -0.26051297783851624, 0.18741485476493835, 0.14522407948970795, 0.7733677625656128, 1.4385430812835693, 0.40826690196990967, -2.6148111820220947, -1.4702372550964355, 0.3964575231075287, 0.189729243516922], [0.5647487640380859, -0.4691148102283478, -0.8390646576881409, -2.119741439819336, 1.2697396278381348, -0.17913271486759186, -0.012074953876435757, 0.8081372976303101, 0.424050509929657, -1.169085144996643, -0.04846527427434921, -1.700931429862976, 0.5132476687431335, 0.3563413619995117, 0.3405965566635132, -0.11801651865243912, -0.6041409373283386, -1.4528546333312988, 1.2263071537017822, 1.3986080884933472, -1.8026334047317505, 1.262807846069336, -0.39605799317359924, 1.1857118606567383, 0.37867745757102966, 0.07788706570863724, -0.8049224615097046, 0.9496068358421326, -0.07573398947715759, -0.26977959275245667, -2.0265212059020996, 1.411461353302002, -0.5114925503730774, 1.3836973905563354, 1.2021013498306274, 0.749608039855957, 1.1505497694015503, -0.008827163837850094, 1.957452416419983, -0.927223265171051, -0.049312446266412735, -0.9200973510742188, 1.2908564805984497, 0.5107924342155457, -1.6432687044143677, 0.46641016006469727, -0.8996498584747314, 0.36936789751052856, -0.5529613494873047, -0.4179026782512665, -0.0038169475737959146, -1.763211727142334, -1.4787425994873047, 0.8576810956001282, -0.7199608087539673, -1.6171027421951294, -0.33151355385780334, -0.5633963942527771, 1.2076127529144287, -1.2096877098083496, 0.4218611717224121, 1.9306038618087769, -0.07640667259693146, -0.3393857479095459, 1.137536644935608, 0.7252888083457947, -0.9137760996818542, -0.0448981374502182, 0.12066970020532608, -0.7308655381202698, -1.2987067699432373, -1.0659195184707642, 1.9390053749084473, -0.31480517983436584, 0.2966500520706177, 1.7534304857254028, 0.4642014503479004, -1.4546979665756226, -1.1487960815429688, 1.3570466041564941, 0.7853780388832092, 1.0358108282089233, -1.2498401403427124, 0.3628290593624115, -0.4411779046058655, -0.08141235262155533, -0.5220603346824646, -0.2501715421676636, -0.9225900173187256, -0.16762197017669678, -0.02523145079612732, 0.33306244015693665, -0.6311191320419312, 0.6475819945335388, -0.4332675635814667, -0.61371910572052, 1.0635082721710205, 1.9768882989883423, -1.8447105884552002, 1.7256076335906982, 0.6751457452774048, -0.5249210000038147, -0.273090124130249, -0.23542393743991852, -0.3664928078651428, 0.5254719853401184, 0.2535121738910675, -1.0169936418533325, -1.7334189414978027, 0.7157883048057556, 1.1003113985061646, -0.7587859034538269, 0.4361893832683563, 0.8756173849105835, -1.7375901937484741, -0.06461154669523239, -1.296583890914917, 0.836936891078949, -1.3069474697113037, -0.9420373439788818, -0.23078875243663788, 0.2677282989025116, -1.2187206745147705, -0.7127789855003357, -0.18275976181030273, 1.6987073421478271, -0.5175496339797974, -0.15004238486289978], [-2.5340638160705566, -0.8149845600128174, -0.5489063858985901, -0.6109864711761475, 0.44678589701652527, -1.2176218032836914, 0.17137308418750763, 0.7398212552070618, -0.775063693523407, -0.07894667237997055, -0.2536522448062897, -0.10857672989368439, 1.1580547094345093, 0.1339738816022873, -1.72248375415802, -1.1829816102981567, -0.20083017647266388, 0.8591976761817932, 1.0517629384994507, -0.144046351313591, -1.2951463460922241, -0.026379121467471123, -0.016478614881634712, -1.2246863842010498, -0.885138213634491, 0.205352783203125, -0.4452736973762512, -0.4141717851161957, 1.6796029806137085, 0.23770877718925476, 1.5679435729980469, -1.2620506286621094, -1.0460749864578247, -0.38608357310295105, -0.30543243885040283, 2.005242347717285, 1.1625664234161377, 0.6157835721969604, 1.3757354021072388, -0.5845698118209839, 0.19595003128051758, 0.15250125527381897, -0.3216859698295593, -0.5251495838165283, 1.0082695484161377, 0.018182259052991867, -0.028862107545137405, 0.2134290635585785, 1.2442928552627563, 0.223025381565094, 0.4957161247730255, -1.6944046020507812, -0.5217873454093933, -0.18658433854579926, -0.1954277753829956, -1.2748225927352905, -0.03380519896745682, -1.0739898681640625, -1.0639915466308594, -0.5408577919006348, 0.7708869576454163, -0.5447354912757874, 0.33681705594062805, -0.20161844789981842, 1.0049673318862915, -0.24751190841197968, 0.26000043749809265, 0.9496336579322815, -0.5557637214660645, -0.8327792882919312, -2.10022234916687, -0.5718466639518738, -0.012780806981027126, -1.4900100231170654, -1.2709534168243408, 0.679990291595459, -0.9902591109275818, 1.8144391775131226, -2.1947884559631348, 1.0572689771652222, -0.6368069648742676, 0.5790994167327881, -0.03510995954275131, -0.14353783428668976, -1.6921075582504272, 0.13229551911354065, 0.14111341536045074, 0.277458131313324, 1.3127574920654297, -0.8951829075813293, 0.9614686965942383, 0.7534445524215698, -1.2451050281524658, -2.062809467315674, -1.4377448558807373, 0.05738350376486778, -0.7689782977104187, -1.8260242938995361, -0.7967139482498169, 0.4389902949333191, 0.7973684668540955, 0.7953158020973206, -1.3958559036254883, 0.21740350127220154, 0.5988420248031616, 0.848766028881073, -0.8719494938850403, -0.07197015732526779, 1.2642501592636108, -0.6917684078216553, -0.9135850667953491, 0.8146741390228271, 0.26252317428588867, 0.02917022444307804, 1.0046879053115845, -0.1627105474472046, -0.8186177015304565, 0.7289788722991943, 0.9089086055755615, -0.7901278138160706, 1.3517146110534668, -0.6752758622169495, -2.366176128387451, 0.05287463217973709, -0.32071617245674133, -0.6696258783340454, 0.9338557124137878, -0.8281172513961792], [1.1537131071090698, -0.8214334845542908, 0.06898725032806396, 0.6933797597885132, 0.448262095451355, -1.7299482822418213, 0.13762864470481873, 0.7294104099273682, -1.6284984350204468, -0.7256157994270325, 1.091052770614624, 0.2865392565727234, -0.17985492944717407, 1.1224521398544312, -1.0712755918502808, 1.1767151355743408, -0.622027575969696, 0.673751950263977, -0.06846880912780762, -0.5406374335289001, 1.2267588376998901, -2.519467353820801, -0.1818917989730835, 0.7985368371009827, -0.15369892120361328, 0.9588105082511902, -1.0868195295333862, 0.8548958897590637, 0.27978166937828064, 0.10981008410453796, -0.3730566203594208, 0.5706210732460022, 1.027368426322937, -0.5763455629348755, -0.5436042547225952, 0.23832197487354279, -0.2601872384548187, -0.445361852645874, -0.8559513092041016, 0.5379806160926819, -0.6268777251243591, -0.24823762476444244, -1.0939782857894897, -0.6076329350471497, 1.809512734413147, 0.6227036714553833, 0.6464993357658386, -0.20013666152954102, 1.75216805934906, -0.1769828349351883, 0.7658352851867676, 1.4140266180038452, 1.0224348306655884, 0.13357047736644745, 0.13514123857021332, 0.39225098490715027, -2.686119794845581, -0.43974295258522034, 2.0519564151763916, -0.26062098145484924, -0.09811794757843018, -0.711449384689331, -1.6468017101287842, -0.29321742057800293, 1.4285626411437988, 0.49557873606681824, 1.6826932430267334, 1.8356053829193115, 0.43078017234802246, 0.08932338654994965, -0.03637261316180229, 0.02042151428759098, -0.4157266318798065, 0.8391491174697876, 0.8686219453811646, 1.6019333600997925, 0.3236345946788788, 0.7556440830230713, 1.0364692211151123, 0.7695239782333374, 0.34338924288749695, -0.38470974564552307, 0.5354427695274353, -0.0054364511743187904, 0.11706189811229706, -1.1739991903305054, 0.283419132232666, -0.3237035274505615, -0.4002525806427002, 0.7519704699516296, -0.07385685294866562, -0.17563462257385254, -1.7664148807525635, -0.0005640905001200736, 0.7628870010375977, -1.6183923482894897, 0.16465218365192413, 1.0935261249542236, -1.1340421438217163, -2.9628634452819824, -0.2793869972229004, 0.04623159021139145, -1.122818946838379, 0.6981483101844788, 0.699845016002655, 1.138440489768982, 0.6740210056304932, 0.47798505425453186, -0.6899049282073975, -1.0793479681015015, -0.6024608016014099, 0.14050398766994476, -1.0342577695846558, 0.7822548151016235, 1.6511837244033813, 0.5091966390609741, 0.9564719796180725, -0.9003150463104248, -0.21306617558002472, -1.9092711210250854, -0.6111341714859009, -0.4941995441913605, -0.4033162295818329, -1.29049813747406, -1.5176790952682495, 1.4755504131317139, -1.7957780361175537, -1.0952702760696411], [-0.4409048855304718, -0.802818238735199, -0.07083680480718613, 0.09511970728635788, -2.4634206295013428, -0.7842642664909363, 1.0603991746902466, 1.337828516960144, 1.2433466911315918, -1.2837647199630737, 1.6227426528930664, 1.831504225730896, -0.19468234479427338, -0.8882285356521606, -0.7791088223457336, 1.9049619436264038, -0.23489248752593994, 0.02446192130446434, 1.1406184434890747, 2.6256861686706543, 0.10495420545339584, 0.7423610091209412, -0.22607015073299408, -0.3105258047580719, 0.9301168918609619, -0.5480623841285706, -0.10337946563959122, -0.6838639378547668, 1.00733482837677, 1.1180602312088013, 2.491546392440796, 1.1855207681655884, -3.0788888931274414, 0.2997075319290161, -0.8770543336868286, 0.7946668863296509, -0.5445777177810669, -1.5219519138336182, -0.70379638671875, 0.44763466715812683, 1.3770267963409424, 0.8974094986915588, 0.2585412859916687, 0.7236241102218628, -0.9508488178253174, 2.4116218090057373, -0.12303994596004486, 0.15382201969623566, 0.6383587718009949, -0.6828066110610962, -0.12299016863107681, 1.0802878141403198, -0.6828979849815369, -0.20913715660572052, -0.5828081965446472, -0.04638942331075668, -1.38335120677948, -1.4280409812927246, -0.2224251925945282, 0.4422013759613037, -2.142899751663208, -1.5510445833206177, -1.2718790769577026, 1.146885871887207, 1.3627811670303345, 1.494302749633789, -1.071700930595398, -1.7042745351791382, 1.2711172103881836, -1.1329782009124756, 3.0562584400177, 0.8443110585212708, 0.4495997130870819, 0.45460808277130127, -1.6980482339859009, 0.44318726658821106, -0.37167468667030334, 0.1631316840648651, 0.46878355741500854, 0.6695973873138428, 0.009381284937262535, -0.3785476088523865, -0.08118081092834473, -0.6501216292381287, 1.0751397609710693, 0.3405710756778717, -0.13613556325435638, 0.22249987721443176, 0.40689951181411743, 0.6158170700073242, -0.38826656341552734, -0.11839767545461655, -0.797713577747345, 0.06850734353065491, -1.1780654191970825, -0.2784372568130493, 0.9629256725311279, 0.7427963018417358, -0.4801499843597412, 0.9539660215377808, 0.3140435516834259, -0.09210389852523804, 0.7270525097846985, 0.440573126077652, 0.6451762914657593, 0.5469509363174438, 0.6990097165107727, -1.044190764427185, -0.6321289539337158, -0.9222623705863953, -1.8834733963012695, 0.32164499163627625, -0.09314499795436859, 2.264159917831421, -0.9958967566490173, 1.7000489234924316, -1.2203466892242432, -0.29607436060905457, -0.04713127017021179, -0.5520991683006287, 0.15771709382534027, 1.146485447883606, 0.36235037446022034, -0.4978421926498413, -0.10623997449874878, 0.18152859807014465, -0.13513413071632385, 1.3288427591323853], [-0.10484594851732254, -0.5848520994186401, 0.5457186102867126, -0.8033501505851746, 0.908845841884613, -1.0330654382705688, 0.5302083492279053, -0.203756183385849, 0.17368508875370026, -0.41460639238357544, -0.7032063007354736, -0.0347287654876709, 0.8042054772377014, 0.32750898599624634, 0.30309802293777466, 0.0884513407945633, -1.0492781400680542, -0.8291218876838684, -0.2675646245479584, 0.0861441120505333, 1.4375637769699097, -0.6310997009277344, -0.8229957818984985, 0.4032179117202759, -1.1628118753433228, -1.5584676265716553, -0.29009655117988586, 1.5292537212371826, 2.1050384044647217, -0.8551775217056274, 1.2118228673934937, -0.9586862921714783, 0.1816631406545639, -0.5249228477478027, -1.0529890060424805, 1.772233247756958, -1.7115148305892944, 1.0914993286132812, 0.63178950548172, 0.34353217482566833, 0.9509280323982239, 0.9796493649482727, 0.7239536643028259, -0.3848629295825958, 0.41539323329925537, 1.8592900037765503, -0.06720074266195297, -1.5791620016098022, -0.6104216575622559, -0.9443185925483704, -1.7213937044143677, 0.3472004234790802, -0.2724219262599945, 0.24057000875473022, 0.5556604862213135, 0.8386520147323608, 1.3633708953857422, 1.9177110195159912, -0.19967341423034668, -0.1526254266500473, -1.0133124589920044, -0.31516847014427185, -0.3570614159107208, -0.6204875707626343, -0.5842490792274475, -0.45853808522224426, 0.5329084396362305, -0.8891956806182861, -0.10808980464935303, -0.1995847225189209, -0.7119032740592957, -0.24364018440246582, -1.0410484075546265, -0.507243275642395, -0.038987141102552414, 0.431710422039032, 0.6877015233039856, -0.5859867334365845, 0.3089711368083954, -1.9925432205200195, -1.3058420419692993, -0.7863225340843201, 0.10991831868886948, 1.3389989137649536, 1.7216320037841797, 0.4420918822288513, 0.5105917453765869, 1.167235255241394, -0.3446590304374695, 1.7130937576293945, -0.22212615609169006, -1.3062809705734253, 0.24381446838378906, -0.4561930298805237, -1.6714965105056763, -1.197152853012085, -0.3191913068294525, -0.9587498307228088, -1.1825973987579346, 0.5734640955924988, -0.014049068093299866, 0.8029009103775024, -0.04347076639533043, 0.6479301452636719, 0.11372704803943634, 0.5750625729560852, -0.5987765789031982, -0.7839053869247437, -0.3763796091079712, 0.7103636264801025, 0.4203568696975708, -0.12413713335990906, 0.8840983510017395, 0.562468945980072, 0.4179477095603943, -0.47161078453063965, 1.9780397415161133, -0.6467087864875793, 1.0970368385314941, 1.460542917251587, 1.3274667263031006, 1.8506721258163452, 0.13906919956207275, 0.5607755780220032, -0.02330969274044037, -1.0242098569869995, -1.04104483127594, 1.059822916984558], [-1.2678630352020264, 0.6217015385627747, -0.5400785803794861, -1.4564152956008911, 1.6682401895523071, 1.3298449516296387, -0.17047028243541718, 2.950575351715088, -1.6510051488876343, -1.3767659664154053, 2.0161547660827637, -1.1083954572677612, -1.0073550939559937, -0.6334421038627625, -1.2770169973373413, -1.4027901887893677, 1.5181821584701538, -0.20750977098941803, -0.18683166801929474, -2.8832223415374756, -1.521519422531128, -1.2885373830795288, -0.8300071358680725, 0.15269190073013306, -0.8700466752052307, 0.5832368731498718, 0.8532834649085999, 1.6266086101531982, -0.479799747467041, -0.031322989612817764, 0.38159722089767456, 0.655659019947052, -0.5280676484107971, 0.6407076716423035, -0.7507140636444092, -0.9654656052589417, 0.09529805183410645, -0.6677600145339966, -0.5281347632408142, 1.049166202545166, 1.4578266143798828, -1.5840272903442383, -0.688804030418396, -0.22157752513885498, 0.4841049909591675, 0.38572561740875244, -0.40217533707618713, -0.8945457339286804, -0.37185147404670715, -1.4080275297164917, -1.6235945224761963, 0.05435623973608017, -0.0771767869591713, -0.45580312609672546, -0.6695565581321716, -1.821336030960083, 1.1759467124938965, 1.2152272462844849, -1.2245835065841675, -1.0456006526947021, 1.184942364692688, -0.38030409812927246, 0.14676660299301147, -0.7532799243927002, -0.5956568121910095, -0.5207935571670532, -0.4406448304653168, -0.9897651672363281, 1.8232669830322266, 0.9119613766670227, 1.8210991621017456, 0.2791837453842163, 1.5820549726486206, -1.2853158712387085, 0.5464402437210083, -0.5833737850189209, 0.07485183328390121, 1.4580082893371582, 1.6136901378631592, -0.3416074514389038, -0.2565461993217468, -0.8646761775016785, -0.44170260429382324, 1.5784326791763306, -0.3130544126033783, -0.7246617078781128, 1.0283267498016357, 0.2871563136577606, -1.0448731184005737, 2.182056188583374, 0.08967363834381104, -0.7906382083892822, 0.35043802857398987, -0.7153213024139404, -1.451269507408142, 0.9128398299217224, -0.4476446807384491, 0.7580687999725342, -0.8943117260932922, -0.26933789253234863, 0.7155206203460693, 0.0815860703587532, 0.014376489445567131, -0.6399487257003784, -0.9557005167007446, -1.4958748817443848, 1.3296297788619995, -0.5790818929672241, 0.35661789774894714, -0.08167485147714615, 1.8876582384109497, 0.6138163805007935, -1.5333325862884521, -0.32587969303131104, -0.6442775726318359, -1.3364571332931519, 0.10491408407688141, 0.47815313935279846, -0.3769875764846802, -1.2981081008911133, 0.6906461119651794, -0.055901240557432175, 1.838314175605774, 0.8228186964988708, 0.5144926905632019, 2.023055076599121, 0.4457109868526459, 1.4525456428527832], [-1.2631549835205078, -0.7290568947792053, 1.109479546546936, 0.28115448355674744, 0.7114424109458923, 0.268146276473999, -0.9324892163276672, 0.24901506304740906, 0.6658227443695068, 0.7451605200767517, 1.0171027183532715, -1.020504355430603, -2.2016942501068115, 1.0608105659484863, -1.6726243495941162, 2.2488582134246826, 0.23266805708408356, 0.6456114649772644, -0.18634308874607086, 0.03657511994242668, 0.8202539682388306, 0.3101288378238678, 0.5609868168830872, -0.8717619776725769, 0.4718817174434662, 0.16809414327144623, -0.36235326528549194, -0.7393373250961304, 0.19326288998126984, -1.5348128080368042, 0.22201761603355408, 0.07665517926216125, -1.497910976409912, 1.4104045629501343, -1.5668596029281616, 1.0105122327804565, -0.32141542434692383, -1.4774284362792969, -1.2009459733963013, -0.18164965510368347, 0.03167860582470894, 0.1759156733751297, 0.16426421701908112, -1.4431096315383911, -1.1364878416061401, 2.150275945663452, -0.015121678821742535, -2.1155381202697754, -0.44926637411117554, 0.44466596841812134, -0.48773330450057983, 1.552164077758789, -0.14180774986743927, 0.848683774471283, 0.41981634497642517, 0.8093540668487549, 0.3471565842628479, -0.19982154667377472, -0.7341407537460327, 0.11055009812116623, 0.17826931178569794, -0.45368435978889465, 0.3977830410003662, -2.1208231449127197, 0.1723557561635971, 0.7699755430221558, 0.7328338623046875, -0.39020949602127075, -0.14355918765068054, 0.29692474007606506, -0.5637650489807129, 0.9793065190315247, -0.16975577175617218, 0.09369809925556183, 1.0344773530960083, -1.0040631294250488, -2.2770888805389404, 0.580799400806427, -0.5422090291976929, 0.03314204514026642, -0.07766694575548172, 0.1593094766139984, -0.17231319844722748, -1.4690991640090942, 0.005572488531470299, 0.7725745439529419, 0.3638902008533478, -1.0541841983795166, -0.07645486295223236, 0.5959997177124023, 0.663210928440094, 1.617270827293396, -1.4265692234039307, -0.0209483802318573, 0.9441742897033691, 0.41338402032852173, 0.9609470963478088, 0.19349287450313568, 0.9610134959220886, 0.40530577301979065, -0.31331518292427063, -1.5384997129440308, 1.2138038873672485, -1.8453627824783325, 1.1509546041488647, -0.4950522184371948, 1.0027575492858887, -0.05854137986898422, -0.8330298662185669, 0.6019905805587769, 0.07326171547174454, -2.695246458053589, 0.9367987513542175, 0.09045889228582382, -0.3108390271663666, -0.6778140068054199, 0.18133874237537384, 0.07569149881601334, -0.44282838702201843, -0.22895564138889313, 0.2667347490787506, 0.19408603012561798, 1.0102325677871704, -0.7364428639411926, -0.1207922026515007, 0.22211681306362152, -1.5737693309783936, -1.3104909658432007], [0.008907350711524487, 0.001389614655636251, -1.3929463624954224, 2.3747494220733643, 1.8401519060134888, 0.5591205358505249, -1.2027629613876343, -0.3268950581550598, -0.3893776535987854, -0.0033864551223814487, 1.1649483442306519, -0.2951001822948456, 0.17721496522426605, -0.412166029214859, -0.6951320171356201, -0.40600892901420593, 0.32715529203414917, -0.06801891326904297, 0.5010573267936707, 1.151578664779663, 0.6407126784324646, -1.0056850910186768, 0.1489885002374649, -0.30898475646972656, 0.22427302598953247, 1.0947227478027344, -0.7532221674919128, 0.7383251190185547, -1.713887095451355, -0.4850936830043793, -1.5556433200836182, 0.1590839922428131, 1.7831528186798096, 1.317984938621521, 0.5862501859664917, 0.3820306658744812, 0.778914749622345, -0.22092317044734955, -0.404477596282959, -0.9695398211479187, 0.0014915455831214786, -0.22476829588413239, -1.3718715906143188, -1.5833806991577148, -0.10011056065559387, 0.6737661361694336, 0.24593843519687653, -0.8502050042152405, 1.3459093570709229, 0.4795570969581604, -2.7315053939819336, 0.8054850101470947, -1.6093617677688599, 1.5879764556884766, -0.49983784556388855, -1.2780413627624512, -0.154579296708107, -0.2156052589416504, -0.37768876552581787, 0.4430592954158783, 1.499009132385254, -0.799486517906189, 0.013947845436632633, -0.7868545651435852, 0.30095675587654114, 0.40289008617401123, 0.5931687951087952, 0.5962008237838745, 0.7292695045471191, 0.7585128545761108, -1.9123585224151611, 0.216949462890625, -0.34996655583381653, 0.6206392645835876, 0.36339741945266724, 1.330491542816162, 1.1735773086547852, 0.9830071330070496, 0.5779833197593689, -0.7419430613517761, -1.6198173761367798, -0.7616747617721558, 0.6537314653396606, 0.6208292841911316, 0.4590142071247101, 1.9043378829956055, -1.927088975906372, 1.2865595817565918, -0.22851777076721191, -1.7230451107025146, 0.37327510118484497, -0.19403570890426636, -0.8668729662895203, -0.8682513236999512, 0.8253288269042969, -0.17337538301944733, 0.4860597252845764, -0.7183646559715271, 0.10971364378929138, -0.48875993490219116, -1.4078627824783325, -0.15828581154346466, 0.36490654945373535, 0.8657721281051636, -1.266893982887268, -1.756813406944275, -0.6534087657928467, 0.9121459722518921, 0.07423454523086548, -0.7674898505210876, -0.47461143136024475, -1.9447708129882812, 0.29794400930404663, -1.1862068176269531, -0.32296615839004517, 0.32690995931625366, -0.16133859753608704, -0.08190622925758362, -0.2642751932144165, -0.31408652663230896, 2.1604137420654297, -1.5197571516036987, -0.11134491860866547, 0.10070446133613586, 1.10098397731781, -1.47666597366333, -1.8753471374511719, -1.2428890466690063], [0.7194961905479431, 0.31953907012939453, -1.2798162698745728, -1.5386570692062378, -0.22162175178527832, -0.5562637448310852, 0.5700831413269043, -0.22505201399326324, -0.9814375042915344, 1.392926812171936, -1.7545166015625, -1.2356703281402588, 1.3726260662078857, 1.1746084690093994, -2.2086679935455322, -0.49576741456985474, -0.6523581743240356, 0.726091206073761, 1.9745320081710815, -1.408289909362793, 1.1451491117477417, 0.5205678343772888, -1.4770524501800537, 0.007749434094876051, 0.5546969175338745, 1.1816356182098389, -0.5208981037139893, -0.6933255791664124, -0.16940519213676453, -0.45204588770866394, -0.34108489751815796, -1.138223648071289, -1.7012264728546143, -0.7849201560020447, -0.18818879127502441, -1.6370495557785034, -2.1748757362365723, 0.8269021511077881, 0.29915666580200195, 0.007358182221651077, -0.9420587420463562, 0.1106170266866684, -0.22421179711818695, -1.036242127418518, -1.180783987045288, -0.7767730355262756, -0.2530783712863922, 0.9635331630706787, -1.3609874248504639, -0.9227635264396667, 0.9936718344688416, 1.3860387802124023, 0.9046016335487366, 0.896952748298645, 0.47218945622444153, 0.33690622448921204, 0.7093927264213562, 0.26571813225746155, -0.07664217054843903, 2.3092117309570312, -0.8486781120300293, -0.7196037769317627, -0.39003366231918335, 0.06562858819961548, -0.20494964718818665, 0.8121962547302246, -1.6749544143676758, 0.02906598523259163, -0.4343198239803314, -0.3024621605873108, -0.29848605394363403, 2.2087862491607666, -0.3948782682418823, -0.17819486558437347, -0.041425641626119614, -2.0513217449188232, -0.9418870806694031, 0.08673076331615448, 0.6347516775131226, 0.477546364068985, -0.5274757742881775, -0.6397243142127991, -0.10228709131479263, 0.9466778039932251, 0.6954299807548523, -0.2176503986120224, 1.030984878540039, 0.9969092011451721, -0.6240046620368958, -1.5596067905426025, 1.4904297590255737, -1.1526530981063843, -0.3537188768386841, -0.27306562662124634, 0.2812182903289795, -1.2574089765548706, 1.9751311540603638, 0.059999242424964905, 0.45839551091194153, 0.49137070775032043, 0.5437594652175903, 0.9880373477935791, -0.07525163143873215, -1.4925473928451538, 2.0333967208862305, -0.966637372970581, 0.30486688017845154, 0.40911152958869934, 1.0250951051712036, -0.21014854311943054, -1.634710431098938, 0.038866590708494186, 1.1089140176773071, -0.48327523469924927, -0.5232459306716919, -0.26443254947662354, 0.7881928086280823, -0.5155506730079651, -0.6412414312362671, 1.0195837020874023, 0.5251656174659729, 1.5057719945907593, -2.2055206298828125, -0.5548801422119141, -0.9019600749015808, 0.0042623789049685, -0.7561963796615601, -0.13001592457294464], [-0.3144509792327881, -0.15611746907234192, -0.4786282479763031, -0.6513181924819946, 1.1505671739578247, -0.6656831502914429, 0.1287253201007843, 2.4050452709198, -1.509123682975769, -0.7111640572547913, 0.3959353566169739, -1.6043288707733154, -2.0980050563812256, -0.47883936762809753, 0.2393008917570114, -0.48422515392303467, 0.12838803231716156, -1.433367371559143, 2.3368585109710693, 0.6836337447166443, 0.04983071610331535, -0.2815100848674774, -1.4494267702102661, -0.6697478294372559, 0.2926102876663208, 0.6778362393379211, 0.6427303552627563, -1.0593986511230469, 0.411667138338089, -0.7382797002792358, -0.5839690566062927, -0.5206815004348755, 0.7902405858039856, -0.6104714274406433, -0.6558908224105835, -0.6871280670166016, -0.8755270838737488, 0.006923696491867304, -0.8118069171905518, -1.5280076265335083, -0.10230569541454315, 0.31990301609039307, 0.26720476150512695, 0.09740635752677917, 0.033455826342105865, 0.25734570622444153, 1.501994013786316, -1.8224053382873535, -0.6856404542922974, 0.5466456413269043, 0.23919238150119781, -0.8020320534706116, -0.2517295181751251, 0.9578768014907837, 0.07165659219026566, 0.8290839791297913, -2.024066925048828, -0.5261850953102112, 2.7579452991485596, 1.2548234462738037, -0.9515603184700012, -0.11122041195631027, 0.26344895362854004, -1.360045075416565, -0.49959537386894226, 0.527316689491272, -1.0872251987457275, -0.4423739016056061, -0.3164981007575989, 0.14364093542099, -0.5641413927078247, 0.23939287662506104, 0.099196657538414, 0.13331197202205658, -0.9984947443008423, 0.07723333686590195, 1.1991583108901978, -0.4026191234588623, 1.375108003616333, -2.481049060821533, 0.7397465109825134, 0.2741962671279907, -0.13649624586105347, 0.44762715697288513, -0.3897719979286194, -0.000516410858836025, 0.6209524273872375, -1.5667147636413574, 0.570853054523468, -0.27055269479751587, 0.7224644422531128, -0.03173994645476341, -0.10706104338169098, -0.15390805900096893, 0.8828096985816956, -1.220197319984436, 0.40074801445007324, 1.3908969163894653, 0.9937834739685059, -0.6837447285652161, -0.273710697889328, -0.2787191569805145, -1.2177138328552246, -0.05264350771903992, 2.1111319065093994, 1.1625458002090454, 0.33644187450408936, -0.23374389111995697, -0.8289977312088013, 0.13047920167446136, 1.627088189125061, -0.11706526577472687, -1.2839105129241943, -1.2676523923873901, -0.3705625534057617, -0.2220739871263504, 2.0443902015686035, -1.7624186277389526, 1.2279518842697144, 0.08120839297771454, 0.9350545406341553, 0.044942859560251236, -0.7900651693344116, -0.8045899271965027, -0.8401784896850586, 2.2750184535980225, 1.1784108877182007, -1.9450033903121948], [0.32931917905807495, 1.9542983770370483, 1.9606257677078247, 1.7840509414672852, -0.758618175983429, -0.6231887936592102, 0.9147258400917053, 1.04344642162323, -1.583311676979065, -0.8068037629127502, 0.6337066888809204, -1.1379600763320923, -1.79814875125885, 0.4331919252872467, -0.35579532384872437, -0.30136460065841675, -2.127206802368164, 0.475726455450058, -0.03387666493654251, 2.003221273422241, 0.4642221927642822, 1.181225061416626, -0.13361269235610962, 0.8029252290725708, -0.12318204343318939, -2.039703845977783, -0.4212327301502228, -1.0938730239868164, 0.25117456912994385, 1.9194327592849731, 0.35672685503959656, -0.5374223589897156, 1.1471620798110962, -0.9319814443588257, 1.4398092031478882, -0.2955101430416107, -1.0493322610855103, -0.6831772923469543, -0.3587028384208679, -0.3063622713088989, -1.547258973121643, -0.17237715423107147, -0.10566477477550507, 2.411165952682495, -1.9827497005462646, 1.1871790885925293, -2.8051538467407227, -0.8327367305755615, -1.1329379081726074, 0.41249382495880127, -0.22621306777000427, 0.5559682250022888, -1.1040780544281006, 0.5196835994720459, 0.5674611926078796, 0.6931515336036682, 0.3146694302558899, -0.9552320837974548, 1.0118457078933716, 2.0737364292144775, 0.9337905645370483, 0.48564234375953674, -1.0282057523727417, 0.6884482502937317, 2.046705722808838, 0.6673883199691772, -0.8895907998085022, 0.0648234561085701, 1.5433101654052734, 0.0061042229644954205, -0.7590191960334778, 0.1461777240037918, 1.1287200450897217, 1.222846508026123, 0.36582151055336, -1.9396803379058838, 0.3398553729057312, -0.7528315782546997, -0.18848508596420288, 0.21933820843696594, 0.46838587522506714, 0.4199278950691223, 1.975728154182434, 1.2790625095367432, 0.20150339603424072, -1.7732617855072021, -1.0943853855133057, 0.365464985370636, 1.157934308052063, -0.2872801721096039, 0.7698266506195068, 0.42654022574424744, 0.007796918042004108, 1.6341009140014648, 0.9396937489509583, -0.4386811852455139, 0.25363221764564514, 0.43085935711860657, 0.33281999826431274, -0.750879168510437, 0.8674307465553284, -0.1654079705476761, 0.5017716884613037, -0.7229197025299072, -1.2544217109680176, 0.1021009013056755, -0.018541933968663216, -1.0175087451934814, 1.0784088373184204, 0.07652289420366287, 0.5654518604278564, -0.7641130685806274, 0.7461007237434387, -0.012329674325883389, 1.9596010446548462, 0.031915564090013504, -1.9870188236236572, 0.24086681008338928, 0.5600166320800781, 1.0174064636230469, -1.1981416940689087, -3.221674680709839, -0.18877071142196655, 0.35958510637283325, 0.3768942952156067, -0.7766764163970947, -3.6211509704589844, -0.3970705568790436], [-1.7843095064163208, 0.02716774493455887, -0.8300690650939941, -0.3426949679851532, 1.314971923828125, -1.152303695678711, 0.1287381649017334, -0.43803462386131287, -2.413268804550171, 0.3298686742782593, -1.4248807430267334, 1.2641621828079224, 0.49852660298347473, -0.5192607641220093, -0.4265824556350708, -0.3944152891635895, 0.07207644730806351, 0.3503348231315613, -0.5006691217422485, 0.2491789609193802, 0.9052947163581848, 0.5806366801261902, 0.6741019487380981, 2.684654712677002, -0.01988319307565689, -1.279380202293396, -0.6936812400817871, 0.20188945531845093, 0.36345162987709045, 0.9018676280975342, -0.672625720500946, 0.5869889855384827, 0.1986045390367508, 1.0755741596221924, -0.6434378027915955, 0.43693774938583374, 1.877028226852417, -0.4380623400211334, 0.6254349946975708, -0.8335393071174622, 0.33719176054000854, 0.7879917621612549, 0.4680764377117157, -0.9780993461608887, 0.40487149357795715, -0.07690445333719254, 1.2502270936965942, 1.6306805610656738, 0.4379188120365143, 0.8418063521385193, 0.777419924736023, -0.37774768471717834, -0.6208637356758118, -0.40092766284942627, 1.0469756126403809, 1.1566367149353027, -1.9371486902236938, 0.44710543751716614, -0.3925018012523651, 0.8916487097740173, 1.2026357650756836, 0.20326785743236542, -1.0074046850204468, 0.24596385657787323, 1.0087379217147827, 0.6351093649864197, -0.7489380836486816, 1.006456732749939, -0.5788404941558838, -1.9128191471099854, -1.151976227760315, 0.906060516834259, -0.19479788839817047, -0.5970464944839478, 0.004658891353756189, 1.010036587715149, 0.3852638602256775, 1.4835158586502075, -1.4977056980133057, -0.6068407297134399, 1.0309793949127197, -0.32125478982925415, -0.8033186793327332, 0.12332162261009216, -0.0653616264462471, -0.8624488711357117, 0.6215085983276367, -0.23732398450374603, 0.6109269857406616, -0.1387718915939331, 0.08311927318572998, -0.7400631308555603, 1.978318452835083, 1.041958212852478, 2.854783058166504, 0.1265585720539093, 0.7751715183258057, 0.4663660526275635, 0.8895746469497681, -0.9157994985580444, 2.1730446815490723, -0.6858832240104675, -0.6182947158813477, -1.163622498512268, 0.13291068375110626, 1.766481876373291, 0.28826168179512024, 0.1331554800271988, 1.2862142324447632, -1.2410491704940796, -0.46576109528541565, 0.09758084267377853, 0.34848037362098694, -1.2217323780059814, -0.32598626613616943, -1.4024580717086792, -0.29202353954315186, -0.9436408877372742, -1.3596010208129883, 1.187620997428894, 1.4767731428146362, -0.5495695471763611, -0.3102111220359802, -0.12614712119102478, 0.45497363805770874, -0.8880414962768555, -0.09086278080940247, 0.7396097183227539], [0.32701775431632996, -0.26909497380256653, -0.944037139415741, 0.7130690217018127, 0.6752275228500366, -0.07238710671663284, 0.7599326968193054, 0.7009103894233704, 0.32986345887184143, -0.09129592031240463, -0.7432600855827332, -0.15758271515369415, 0.9840576648712158, 2.1774582862854004, -0.5001078844070435, -0.2972354590892792, -0.9115398526191711, 0.7936503887176514, -0.7365768551826477, -1.2322778701782227, 0.749646782875061, 1.6278221607208252, -0.6035726070404053, -0.14620870351791382, 0.10162460803985596, -0.22900615632534027, 0.28516149520874023, -1.9211689233779907, -1.1862949132919312, -1.171076774597168, -0.19488486647605896, -0.896477460861206, -1.0481475591659546, 0.733799159526825, -0.8177309632301331, 0.8858560919761658, -0.9582267999649048, -1.6057039499282837, 0.3358737826347351, -0.2907602787017822, 2.6449081897735596, 0.25074270367622375, 2.674933433532715, -0.25845932960510254, -0.09044792503118515, 2.7323296070098877, 1.368186354637146, -1.3385781049728394, 1.4915461540222168, -1.965989351272583, 0.6697372794151306, -1.0338151454925537, -1.7714649438858032, 0.6268035769462585, 2.5002458095550537, -0.3006785213947296, 0.5979146361351013, -0.5461506247520447, 0.024341117590665817, -0.7263811826705933, -0.5125162601470947, 1.1865081787109375, 0.22640031576156616, -0.8111147880554199, 0.615929126739502, 0.4045563042163849, 0.9088910222053528, -0.37155547738075256, 2.0438971519470215, 0.14835496246814728, -1.959765076637268, -1.1816328763961792, -0.36282339692115784, -0.65468829870224, 0.7397582530975342, -0.036219753324985504, 0.574531078338623, -1.1067767143249512, -0.7495592832565308, 0.9903547763824463, 0.5518047213554382, -1.1831713914871216, -0.017490018159151077, -0.8745895028114319, -1.2257781028747559, 0.4566520154476166, -1.1098028421401978, 0.6468386650085449, 1.7511513233184814, 1.3499139547348022, 0.6364076733589172, 0.2878701984882355, 0.057392556220293045, 2.4459216594696045, 0.39837461709976196, 0.7233695387840271, 2.6825766563415527, -0.4870585799217224, 2.3319602012634277, -2.050835132598877, 1.1469639539718628, -0.7512837648391724, -1.2289901971817017, 1.6615132093429565, -0.4111827611923218, -0.7990343570709229, 0.2656899392604828, 1.1986522674560547, 0.5440614819526672, -0.8446000218391418, 0.6553343534469604, 1.5163356065750122, 0.2743329107761383, 0.9487909078598022, -0.5617547631263733, 0.3205004334449768, 0.6234040260314941, -1.5560599565505981, 0.4691632091999054, 1.016320824623108, -0.001018167706206441, 0.25101083517074585, -1.9755522012710571, 1.7419676780700684, -0.33104097843170166, 0.6173534989356995, -0.8273842334747314, -0.6639267206192017], [0.7019575834274292, -1.6477410793304443, -0.629765510559082, 0.11853467673063278, 0.344313383102417, 0.26038986444473267, 0.36300480365753174, 0.7704533338546753, 1.7081700563430786, -0.14863276481628418, -1.6173073053359985, -0.6753908395767212, -0.7880932688713074, 1.0016041994094849, -0.39088335633277893, -0.46874743700027466, -2.0093021392822266, -0.2984072268009186, 0.2924150228500366, -0.0636005848646164, 0.9272989630699158, -0.5014864802360535, -0.5485890507698059, -1.8182207345962524, 0.8271161913871765, -1.3156615495681763, -1.1388099193572998, -0.19719651341438293, -0.8591446876525879, -1.2672704458236694, -0.8876858353614807, 0.19425809383392334, -0.9172865152359009, 0.8926337361335754, 0.33694103360176086, -1.079779028892517, -1.7107120752334595, -0.616858184337616, 0.07344871759414673, 0.3152521848678589, 0.17109082639217377, -1.7161201238632202, 0.6779198050498962, 0.2086045742034912, -0.15613673627376556, -0.19281232357025146, -0.7545937299728394, 0.04077441245317459, 1.589816927909851, -0.3197034001350403, -0.21937398612499237, 0.3882751166820526, -0.31906238198280334, 0.9127013683319092, 1.439117431640625, -0.26611796021461487, -1.004560112953186, 0.33266767859458923, 2.5041420459747314, 0.8787223100662231, -1.0514650344848633, 0.027019549161195755, -0.4385063648223877, -0.5443044304847717, -0.5735507011413574, 0.8760265111923218, 1.9105157852172852, -0.6542444825172424, -0.6756240725517273, -0.3945128619670868, 1.421706199645996, 0.43965309858322144, 0.4965650737285614, 1.470096468925476, 0.12265461683273315, 0.8552051186561584, 0.455230176448822, 0.12601202726364136, -1.0029913187026978, 0.4065692126750946, 0.2941070795059204, -0.4517955183982849, -0.8286258578300476, 0.08847105503082275, 0.28299885988235474, -0.9391708970069885, -0.463186115026474, 0.021930227056145668, -1.6649442911148071, -0.6824023127555847, 1.1233553886413574, 0.46961531043052673, 0.2970540225505829, 0.1330859214067459, 1.1537725925445557, -1.7277239561080933, -4.363180637359619, 0.7762138843536377, 1.1679248809814453, -1.1498661041259766, -0.3513847887516022, -0.3818649351596832, 1.9695305824279785, -0.5861383080482483, -0.3523280918598175, -1.3454480171203613, 0.37217584252357483, -0.7337340712547302, 1.3568129539489746, 0.5165210366249084, -1.4728260040283203, 1.399782657623291, -1.2608498334884644, -0.6936841011047363, 2.1965816020965576, 0.2605055570602417, -0.05104450508952141, 0.011770006269216537, -2.4353559017181396, 0.22106710076332092, -0.34128203988075256, 0.18727609515190125, 2.5104238986968994, 0.9386724829673767, 1.9511295557022095, -0.9544048309326172, 0.9150396585464478, -0.5757725238800049], [-1.1473225355148315, -0.9700369834899902, 1.322807788848877, 0.9407771825790405, 0.38245439529418945, 0.03696088492870331, 1.666276216506958, 0.13099175691604614, -0.21345698833465576, -0.2774946391582489, -0.9772651195526123, -0.05021796002984047, -0.7951170206069946, -1.697959303855896, 0.8848859071731567, -0.2581283748149872, -0.7475817799568176, 0.5724964141845703, 0.5553454160690308, -0.25553077459335327, 0.26486894488334656, -0.7128177285194397, 0.9192713499069214, 0.6148650646209717, 2.020744800567627, -0.9796350598335266, -1.3017901182174683, 1.4855761528015137, 2.336380958557129, 1.2068241834640503, -0.7073491811752319, -0.018063370138406754, -1.5078978538513184, -1.200910210609436, -0.9941104054450989, 1.0411914587020874, 1.4400320053100586, -0.4054880738258362, -0.05329994857311249, 1.2199859619140625, -0.09116343408823013, -1.3185523748397827, -0.17243075370788574, -2.0423665046691895, 0.5697525143623352, -1.7290418148040771, -0.947207510471344, -0.5666529536247253, 0.018449952825903893, 0.14692942798137665, 0.8569585084915161, -1.2687146663665771, -0.215373694896698, -0.3872678279876709, 1.80897057056427, 1.7263472080230713, -0.3834628760814667, -1.049532175064087, -0.03187020868062973, 0.9560542702674866, 1.4158953428268433, -1.495805263519287, -0.27788427472114563, -0.44576239585876465, 0.161427840590477, 1.5851497650146484, -0.16153714060783386, -0.08670248091220856, -1.2214539051055908, -1.169791579246521, -0.9790870547294617, -1.1451400518417358, -1.2123788595199585, -2.7976162433624268, 1.1818900108337402, -1.2371288537979126, 0.11325449496507645, 0.00836271047592163, -0.8406289219856262, -0.4983906149864197, -1.5388195514678955, -0.4971727132797241, -0.32625630497932434, 1.7490633726119995, 0.7983509302139282, 0.6453620791435242, 0.3933796286582947, 0.31251999735832214, 0.5456317663192749, 0.4763621389865875, -0.457207053899765, -1.8497352600097656, 0.3026266396045685, -0.2652723789215088, -0.829047441482544, 0.4230315089225769, -0.7864193916320801, -1.136102557182312, -1.1882010698318481, 1.1108382940292358, 1.3939578533172607, -1.3389707803726196, -1.9402127265930176, -0.7001751661300659, 2.476778984069824, -0.2690173387527466, 1.3537153005599976, -0.36357229948043823, -1.9478309154510498, -1.512387990951538, 1.4396666288375854, 1.2828385829925537, 1.0301344394683838, 1.5245953798294067, 0.36805614829063416, -1.7784905433654785, -2.1998167037963867, 0.1762360781431198, 0.5119538903236389, 0.6924895644187927, -0.7233914136886597, -0.688367486000061, -0.876723051071167, -0.9018690586090088, -0.1906796544790268, -0.19342753291130066, -1.47585928440094, 0.11714322119951248], [0.3011878728866577, -0.33360904455184937, 1.08119797706604, -0.9442914128303528, 1.0695685148239136, -1.1879514455795288, -0.9988773465156555, 0.05187511816620827, -0.041899487376213074, 0.3221694231033325, 0.21385551989078522, 0.9483530521392822, -0.5630037784576416, 0.7518431544303894, -0.616260290145874, 0.36613941192626953, 0.8374307155609131, 0.6347436904907227, -1.2112361192703247, 1.3535407781600952, -0.752215564250946, -2.4053468704223633, 1.2446612119674683, 1.60148286819458, 0.32146865129470825, 0.2822697162628174, 0.5058732628822327, -0.44992172718048096, 1.1147702932357788, -0.8613519668579102, -0.13822895288467407, -0.540031909942627, -1.2935304641723633, -0.35182517766952515, 0.29677918553352356, 0.9191951155662537, 0.5619639754295349, 0.8680722713470459, -0.012720638886094093, 0.07699166238307953, 0.5265111923217773, 1.0358099937438965, -0.6750857830047607, 1.092695951461792, -1.9318445920944214, -0.9264407157897949, 0.36206817626953125, -0.9474324584007263, -1.3058462142944336, -1.500407099723816, -1.4714603424072266, 0.3688567280769348, -1.387686014175415, 0.6732981204986572, 0.2744807302951813, 0.1772954761981964, -1.7309640645980835, -0.27555516362190247, 0.8145590424537659, 1.2326680421829224, -0.6762927770614624, -1.9093983173370361, -1.2042295932769775, 0.13261622190475464, -0.11443261057138443, -0.581925630569458, 1.184373378753662, -1.0037884712219238, 0.9160827994346619, -0.8764712810516357, -0.5355451107025146, -1.1701191663742065, -1.482689619064331, -0.16386516392230988, 1.5089175701141357, 0.06337329000234604, 0.2060166746377945, 1.3265423774719238, -2.643578052520752, 0.005131034180521965, -1.3870185613632202, -0.9578636884689331, 0.12859506905078888, 0.578380823135376, -0.3399483859539032, 1.0072664022445679, 1.0209640264511108, -0.5801589488983154, -0.21816104650497437, 0.18263478577136993, 2.0449843406677246, -1.5100189447402954, -0.02528277412056923, 1.2881150245666504, -0.023790651932358742, -0.8336032032966614, 0.5422899127006531, 0.6443725824356079, -0.09908830374479294, -0.21472515165805817, -1.0407373905181885, -1.6962655782699585, -0.2937244176864624, -0.43569400906562805, -0.14710400998592377, -0.3952232897281647, 0.46528923511505127, -0.5764231085777283, -1.1528325080871582, 0.22778525948524475, 0.14343078434467316, 1.277442216873169, 2.315412759780884, 0.21699652075767517, -0.6939254403114319, 0.07977060973644257, -1.4837369918823242, -0.1510985642671585, -0.00751509191468358, 1.241200566291809, -0.72365802526474, -0.16661188006401062, 0.14191216230392456, 1.1785814762115479, -0.053844671696424484, 1.099743366241455, -0.865973174571991, 0.9007774591445923], [0.9721095561981201, -0.08810615539550781, -1.1271693706512451, -0.051162127405405045, 1.368597149848938, -0.8742876052856445, -1.3012595176696777, 0.9231998324394226, -0.5732713937759399, -0.6425166130065918, 0.8345516920089722, 0.36450615525245667, -0.44302308559417725, -0.836784839630127, -0.5623120069503784, 0.7332783937454224, 0.6322163343429565, -1.3119819164276123, -2.4966981410980225, -0.19284707307815552, 0.35530927777290344, -1.3764535188674927, 0.632851779460907, 1.5253944396972656, -0.9661116600036621, -0.20473800599575043, -2.4314074516296387, 0.302024781703949, 0.07773459702730179, -0.3441227376461029, 0.6403009295463562, 0.5578630566596985, 0.7208625674247742, 1.3624906539916992, -0.027453213930130005, 0.36359694600105286, 0.07249011844396591, -0.027094103395938873, -0.6949378848075867, 0.3882407546043396, 0.32365384697914124, 0.7483614683151245, -0.39329805970191956, 0.005547969602048397, 1.9732707738876343, -0.751122236251831, -0.7709919214248657, -0.24705854058265686, -1.138005018234253, 0.7747295498847961, 0.4647713601589203, -0.47363001108169556, -1.8875445127487183, -0.7000211477279663, -0.2330876886844635, -1.2787678241729736, 0.9808782935142517, 1.6734354496002197, 0.3792071044445038, -0.10003337264060974, -0.9738906025886536, 0.9888956546783447, 0.6400623917579651, 0.020820893347263336, -1.3120087385177612, -1.3714743852615356, 0.6437838673591614, 0.409626305103302, -0.7501609325408936, 0.04561576992273331, 0.2732195258140564, -0.1318688690662384, 0.34117209911346436, -0.16989666223526, 0.2884717285633087, -0.6466618776321411, 0.18960702419281006, -0.11265356093645096, 0.6787514686584473, -1.3811461925506592, -1.2196509838104248, 1.919972538948059, 0.2833147346973419, -1.0114792585372925, 0.4145660698413849, 0.6383192539215088, -0.9856671094894409, -0.10603199899196625, 0.186941996216774, 0.2864399254322052, 0.5112936496734619, -0.14107699692249298, -2.24186110496521, 0.1404803842306137, 0.9000908136367798, 0.2843363881111145, -0.265092670917511, 0.13296741247177124, 0.5397732853889465, -0.07694122940301895, 0.16509640216827393, 1.6902600526809692, -2.8718669414520264, -0.8429751396179199, -0.903203547000885, -0.8289703726768494, 1.025881052017212, -1.5584242343902588, -0.5519375205039978, 0.42886289954185486, -0.9727510213851929, 0.3669168949127197, -1.43017578125, 2.61574649810791, 0.14801929891109467, -0.8312296867370605, -2.4354350566864014, 1.693277359008789, 0.43633198738098145, -1.215092658996582, -0.6630201935768127, 0.4403623342514038, 1.5019068717956543, -0.47686484456062317, 0.5582101941108704, -0.5712785124778748, 0.30513015389442444, 0.14362937211990356], [-0.4112248122692108, 0.24814994633197784, 1.4548407793045044, 1.6971231698989868, 0.6679256558418274, 1.0155504941940308, 0.31531378626823425, -0.24966804683208466, -0.5831143260002136, 0.9399705529212952, -0.8851906061172485, -0.5092325210571289, 1.8363271951675415, 0.06205674633383751, 0.15586915612220764, -0.9531523585319519, -2.675811529159546, -0.055973879992961884, 1.0011351108551025, -0.5969932079315186, 2.3059632778167725, -0.06674331426620483, -0.052095621824264526, -1.0856863260269165, -0.21938028931617737, 1.544010877609253, 0.040614306926727295, 0.041437067091464996, -0.07621201872825623, 1.1013779640197754, 0.02822120673954487, 0.7680954933166504, 1.3148958683013916, -0.18723735213279724, -1.6241672039031982, -0.10666017234325409, -0.7402527332305908, 1.0620167255401611, -0.032534558326005936, 0.4218108355998993, -1.3433144092559814, -0.9484469294548035, 1.2259272336959839, -2.113104820251465, 1.1758699417114258, 1.2983546257019043, 1.3480373620986938, 1.8297873735427856, 1.7359882593154907, -2.084437131881714, 0.19156521558761597, 0.9283189177513123, -0.5895222425460815, 0.1834528148174286, -0.6125339269638062, 1.4637147188186646, 0.4874032735824585, 0.430118590593338, -1.6893850564956665, 1.0286498069763184, -0.07526250928640366, 1.0845203399658203, -0.6121665239334106, -1.4947917461395264, -0.46672043204307556, 0.4602690041065216, -0.05471724271774292, 1.1198034286499023, 0.13748091459274292, 0.8978204131126404, -0.1835738867521286, -3.143779993057251, 0.6281360983848572, -0.6061047911643982, -1.1832233667373657, 0.30608445405960083, 0.3384961783885956, 0.045024141669273376, -0.01427990198135376, -0.7590611577033997, 0.17120301723480225, 2.0206966400146484, -0.8793234825134277, -0.6596152782440186, 0.8350315690040588, 0.6480082273483276, -0.7514789700508118, 0.05825768783688545, -0.6057755351066589, 0.6054394841194153, 0.8369665145874023, 1.6614147424697876, 0.8832195401191711, -0.2298564463853836, 0.12138453125953674, -0.5118896961212158, 0.03942979499697685, -0.2288762778043747, 0.5421682000160217, 0.8065458536148071, -0.8525224328041077, -0.9609179496765137, 1.7702451944351196, 0.6782214641571045, -0.05155254155397415, 0.0335577055811882, 0.7121061086654663, 0.7109271883964539, 1.6941207647323608, -1.5681477785110474, 0.10885678231716156, 1.3516755104064941, -1.3529300689697266, 1.1377732753753662, -0.2818397283554077, -1.1687259674072266, -1.322255253791809, -0.6525996327400208, 0.49060338735580444, 0.448045015335083, -0.3877982497215271, 1.027126431465149, 0.8595350980758667, 1.3352630138397217, -0.348177045583725, -0.6761065125465393, -0.10321822762489319, 0.2950890064239502], [1.3172411918640137, 1.0143866539001465, -1.6018993854522705, 0.8116103410720825, 0.19945183396339417, -0.2842656672000885, 1.4250596761703491, 0.4133400321006775, -0.14525535702705383, -0.07161968946456909, 1.104340672492981, -0.016549307852983475, -0.8359427452087402, 0.6041370630264282, -2.21978759765625, -1.5129343271255493, 1.1820061206817627, 0.505527138710022, -0.5835411548614502, -0.6515536308288574, 0.7799631357192993, -0.5358152389526367, -0.8261777758598328, 0.36681291460990906, -0.6346651315689087, -1.1112114191055298, 0.8130343556404114, -2.1872799396514893, -0.3767746090888977, -0.7662186026573181, -0.21793891489505768, -0.2822437286376953, 1.384151816368103, -0.7719086408615112, -1.8666960000991821, -0.78849858045578, -0.7609601020812988, 0.41012969613075256, 0.2628743350505829, -0.2627599537372589, 0.825769305229187, -2.3263113498687744, 0.15892380475997925, -0.12399177253246307, -0.4561958312988281, -0.9378649592399597, -1.252711296081543, -0.9414933919906616, -0.967195987701416, -1.5398499965667725, -1.7791939973831177, 1.4691258668899536, 1.5106542110443115, -0.5427283048629761, -0.6860626935958862, -0.7375332713127136, 0.4099700152873993, 0.08047256618738174, 0.6212692260742188, 0.5099040269851685, 0.7045825123786926, 0.07769768685102463, -0.6495197415351868, -0.782507061958313, 1.8782762289047241, 0.018131891265511513, 0.8125710487365723, -0.7184548377990723, 0.4351969063282013, -0.8362804651260376, 1.4775546789169312, 0.6456533074378967, -1.3094240427017212, 0.3332495391368866, 0.48307791352272034, -1.8228366374969482, 0.6491621136665344, 0.9366341829299927, 2.464787721633911, -0.08402512967586517, 1.9743759632110596, 1.5183610916137695, -0.11463157832622528, 0.9433620572090149, 1.0370393991470337, -0.37213313579559326, 2.369236707687378, 0.35750705003738403, 0.7965986728668213, 0.08321450650691986, -0.6467335820198059, 0.09072758257389069, 0.702709436416626, 2.4643447399139404, -1.14934241771698, -0.478176087141037, -1.4976871013641357, -0.9453938007354736, -0.6173127293586731, 0.19955068826675415, 0.5063514113426208, -0.31436964869499207, -0.7763786911964417, 1.3539623022079468, -0.38424816727638245, -0.7941713929176331, 0.8588294386863708, -0.39658331871032715, -0.25982484221458435, -0.5176734924316406, -0.17735105752944946, -0.349045068025589, 1.492091178894043, 0.10245489329099655, 1.3177629709243774, -0.4369503855705261, -1.097631573677063, 0.4838120639324188, 0.4817488491535187, -1.3554794788360596, -0.6020026803016663, -0.6974664926528931, 1.2700042724609375, 0.09775584936141968, -0.29592180252075195, 1.9389052391052246, -0.9771836400032043, -2.402766704559326], [-0.3301922082901001, 1.9733892679214478, -0.5015354752540588, -1.0367071628570557, 0.15913647413253784, -1.171155571937561, -0.21828030049800873, 1.0428110361099243, -1.3719314336776733, 0.4243835210800171, 0.5121450424194336, -1.2486580610275269, 1.0812673568725586, 0.7604484558105469, 0.38569486141204834, 0.11925598233938217, 0.015008443966507912, -1.8996652364730835, -1.9013795852661133, -1.338965892791748, -0.119136281311512, -1.3557718992233276, 0.6531410217285156, -1.011011004447937, 1.8478530645370483, -0.8905607461929321, 0.5317725539207458, -0.5364161133766174, -0.29204419255256653, 0.13898026943206787, 0.5160443782806396, 3.9564595222473145, 1.8202687501907349, -0.1510099470615387, -0.6740396618843079, -0.8129040002822876, 0.7506586313247681, 1.5103609561920166, 0.9085155725479126, 0.05458398908376694, 0.041873447597026825, -0.21178559958934784, -0.19054806232452393, 1.484082818031311, 1.0999157428741455, 1.8661895990371704, -1.9476438760757446, 0.2732096314430237, 0.179609015583992, -0.11035309731960297, 0.3429739773273468, -0.02488783933222294, -0.8961191177368164, 0.15986444056034088, 1.3390893936157227, -0.7824077606201172, -0.7013934254646301, 0.9631325006484985, 0.5735061764717102, 2.8503236535470933e-05, -0.537010133266449, 0.10524602234363556, 0.5748583078384399, 0.6229844689369202, -0.5049483776092529, 0.02719581499695778, 0.37217289209365845, -1.471780776977539, -0.6989904046058655, -0.36819955706596375, -0.08155057579278946, -0.11283788830041885, 0.30589762330055237, 0.19865627586841583, -1.0919219255447388, 0.2451746016740799, 1.5637855529785156, -1.1625640392303467, 1.8765751123428345, -0.1585962027311325, 0.39726999402046204, -1.056391716003418, 0.35443395376205444, -0.018748179078102112, -1.023112416267395, -1.734043836593628, -0.3381383419036865, 1.3076809644699097, -0.7521389722824097, 0.06114910915493965, -0.7484266757965088, -0.023225076496601105, 1.1664315462112427, -0.488707959651947, -0.8037422895431519, 0.46860989928245544, 0.9928614497184753, -0.21687756478786469, 1.2008813619613647, 0.3572271168231964, 0.12907999753952026, -0.9228337407112122, 0.5605263710021973, 0.10318867862224579, 0.67204350233078, -1.563900351524353, 0.6322258710861206, 0.2935534119606018, -0.7552167773246765, -0.10930344462394714, 0.5462968945503235, -0.9859604835510254, -1.7435812950134277, -0.22449731826782227, -0.22552092373371124, -0.6316737532615662, -0.3395349085330963, -0.204965740442276, 0.7397465705871582, 1.6356220245361328, 0.17025820910930634, -0.5422412157058716, -0.4981236457824707, 0.7248103618621826, 1.021116018295288, -1.2169290781021118, 2.789095640182495, -0.3143382668495178]], [[0.19789081811904907, 0.5879193544387817, -1.3184376955032349, 0.9903567433357239, 0.8177779316902161, 0.7743068337440491, -0.616085410118103, 0.24398937821388245, 0.6814699769020081, -0.8360717296600342, -0.8079847097396851, -0.24911117553710938, -2.584110736846924, -0.35607394576072693, 1.3370469808578491, 0.005968076176941395, 0.8930312395095825, -0.16932858526706696, 0.5889033079147339, -2.1621131896972656, 0.11773654818534851, -0.999265193939209, -0.02837473526597023, 1.2794667482376099, -0.3515792489051819, 0.522208034992218, 0.6449486017227173, 0.8681407570838928, 0.2528716027736664, 1.8383567333221436, 0.010689656250178814, -0.5534741878509521, 0.4617471694946289, -0.5067040324211121, 1.6526442766189575, -0.7705622315406799, -0.5968149900436401, 0.23670274019241333, 1.0571027994155884, 1.0957690477371216, -0.7773667573928833, 1.860844373703003, -0.03776685893535614, -0.5330691337585449, -1.7346001863479614, -0.9644150733947754, -0.850253164768219, 0.4097408950328827, -1.6264287233352661, 1.0017386674880981, -2.58951735496521, 0.6351816058158875, 0.020115207880735397, 0.08628668636083603, -0.17225423455238342, -1.04156494140625, -1.629131555557251, -0.7910057902336121, 0.24790143966674805, -0.5542173385620117, 1.1951502561569214, -1.2490766048431396, 0.005944846197962761, -1.5909847021102905, -1.4833019971847534, -2.3597724437713623, 0.010565401054918766, 0.9427584409713745, -0.8514484763145447, 1.7857486009597778, 0.4316338896751404, -0.8208759427070618, -0.553451657295227, 0.1255318820476532, 0.29009994864463806, -0.14045293629169464, 0.38556960225105286, 1.7012678384780884, -0.2971183657646179, -1.2090983390808105, -0.10038655996322632, -0.8327974677085876, -1.5389658212661743, -0.10046549141407013, -0.4096376597881317, 0.18366952240467072, 0.6144658327102661, -0.5906768441200256, 1.10517418384552, -1.0896787643432617, -0.7578331828117371, -1.8439793586730957, 1.2020432949066162, 0.77421635389328, -0.2682482898235321, -0.13256016373634338, -0.06278595328330994, 0.6497988700866699, -3.2264528274536133, 1.354777216911316, 0.8329572081565857, -0.041222214698791504, -1.465641975402832, -0.7882899641990662, -0.46053221821784973, 0.2909559905529022, -0.3458474278450012, 1.0092612504959106, -1.213873028755188, -0.37738439440727234, -0.8676493167877197, -0.554970383644104, 1.3163625001907349, 0.7842403650283813, 0.25186988711357117, -0.3316700756549835, -1.28728187084198, 0.49661579728126526, -1.7592672109603882, -1.6063259840011597, 0.6847410202026367, -0.5839448571205139, -0.14227885007858276, -0.8113439083099365, -1.4646013975143433, -0.605222225189209, -0.214421346783638, 0.980857253074646], [-0.08508951961994171, 0.8468018770217896, -0.9215655326843262, 0.4324517250061035, 0.10958800464868546, 0.40081116557121277, -0.6698177456855774, 0.6772059798240662, 0.2692406177520752, 0.7094008922576904, -0.8653149008750916, -0.2205725610256195, 1.2123503684997559, 0.5462415814399719, 2.041062355041504, -0.8509116172790527, -0.7814292311668396, -0.7811484336853027, -0.7087576389312744, -0.29341834783554077, 1.2090344429016113, 0.5622250437736511, -0.4028877317905426, -0.5042749643325806, -0.3570401072502136, -0.6465367674827576, -1.4203382730484009, 0.1204342395067215, 0.7504048347473145, 0.5650613903999329, -0.8962080478668213, 0.01865188032388687, 1.6311016082763672, 0.6299808621406555, 0.13897520303726196, -0.2606738805770874, 1.4086054563522339, 0.858785092830658, -1.6404590606689453, -0.05353628471493721, 0.10384751856327057, -1.0997817516326904, -0.30706796050071716, 1.3444558382034302, 0.21578766405582428, -0.691877543926239, 0.25663575530052185, 0.6315252780914307, 0.059303756803274155, 1.6866846084594727, -0.5763401389122009, 0.18083296716213226, 0.7235840559005737, 1.6254909038543701, -0.10971151292324066, 0.38858792185783386, 0.482840359210968, -1.1834038496017456, 0.1144762933254242, -0.1755613386631012, 0.4615514576435089, -0.14761246740818024, 0.16821293532848358, 0.8172217607498169, -0.534338116645813, -0.437857985496521, 0.9764763712882996, 0.28838518261909485, -0.3015468120574951, 1.804304838180542, 0.9276915192604065, 0.31604084372520447, -0.7191008925437927, -0.17324994504451752, 0.49660536646842957, 0.9338545203208923, 0.38444867730140686, -1.4543023109436035, -0.8098980784416199, -0.8727250695228577, -1.541874885559082, -0.5270071029663086, 0.8050128817558289, -0.12289673089981079, -0.7450040578842163, 1.2699931859970093, 0.3415839970111847, -1.7051149606704712, -0.8107427954673767, 0.5642585754394531, 0.6432709097862244, -0.7485646605491638, 1.4974050521850586, 0.35051074624061584, 0.5080569982528687, -0.006429317872971296, 1.917206048965454, 0.4946308434009552, -0.6162309050559998, -0.5799590349197388, 1.2615399360656738, -0.23702210187911987, -0.06312009692192078, -0.5580889582633972, -0.5563270449638367, -1.281987190246582, 1.4266204833984375, 2.040587902069092, 0.940904974937439, -0.1449957638978958, 0.7725144624710083, 1.6851189136505127, -0.30858156085014343, -0.4666576683521271, -0.12450617551803589, 0.11619818955659866, -0.4165828227996826, -0.35312506556510925, 0.2649346888065338, -1.0044928789138794, -0.328843355178833, -0.4313098192214966, -1.2848094701766968, 1.0049171447753906, 0.6798079609870911, -1.9174407720565796, 1.2538087368011475, -0.05682902783155441], [1.412494421005249, 1.0912654399871826, -1.0501641035079956, -0.6353049278259277, 0.3777904510498047, -2.215938091278076, -0.8418809771537781, 0.4861868917942047, -0.09519325941801071, 1.3596447706222534, -0.5360100269317627, 0.10731713473796844, -0.5131022334098816, 0.9210504293441772, -0.5588303208351135, 0.9774694442749023, -1.0273680686950684, -0.8431276082992554, 0.2475782185792923, -0.023551935330033302, 0.17545075714588165, 0.6848441958427429, 0.1949443817138672, 0.049822766333818436, 2.048088788986206, 0.4214794337749481, 0.4155745208263397, 0.17289644479751587, 1.0117907524108887, -0.6704246401786804, 0.6857106685638428, -0.5311013460159302, -0.4288206100463867, -0.19897526502609253, -0.6451416611671448, 0.9051427245140076, 2.326864004135132, -1.6216577291488647, -2.1349704265594482, 0.8202924132347107, -1.056068778038025, 0.9268821477890015, -1.5651909112930298, -0.5830850601196289, 0.37708476185798645, -1.5810906887054443, -1.7597078084945679, 0.03520108014345169, 1.1828933954238892, 0.5816161036491394, -0.06031837314367294, -0.45286181569099426, 0.18688097596168518, -0.17346037924289703, 0.9420295357704163, 0.4934123158454895, 2.2557592391967773, -0.6194261312484741, 0.8091097474098206, -1.0531013011932373, -1.1142089366912842, -0.46175938844680786, 0.5414065718650818, -0.4438993036746979, -2.272432804107666, -1.283521294593811, -0.8184216022491455, 1.112058162689209, -1.5806604623794556, -0.11318162083625793, 0.34096094965934753, -0.6396588087081909, -0.8604435920715332, -0.5602648854255676, -0.31593167781829834, 0.7131472826004028, -0.2544354796409607, 0.18499648571014404, -0.06666356325149536, 0.0446406751871109, -1.1975020170211792, -1.0685248374938965, -0.9243118166923523, -0.3442256450653076, -0.11481279879808426, -1.7793772220611572, -0.9894614815711975, -1.0612682104110718, -0.9679082632064819, -0.03448386490345001, -0.056022245436906815, 0.5927648544311523, 0.4029534161090851, -1.0957884788513184, 1.2244064807891846, 0.2630225121974945, 0.17647968232631683, -1.1208375692367554, 0.7547159790992737, 1.5344825983047485, -0.11288601905107498, -0.28560081124305725, -0.9969291090965271, -0.8800095915794373, -0.6753993034362793, 0.7804689407348633, 0.4246883988380432, -0.9459370374679565, 1.1158616542816162, -1.1352301836013794, 0.6517556309700012, 1.1879056692123413, -0.6156936883926392, -0.8101810216903687, -0.22147811949253082, -1.2577284574508667, -1.199222207069397, 0.4092104136943817, 0.6459028720855713, -0.6577168107032776, -0.9565438032150269, 0.07948759943246841, -0.36368128657341003, -0.3701601028442383, 1.978932499885559, -1.1445801258087158, -0.712485134601593, 0.22102971374988556], [-0.6817045211791992, 0.7484190464019775, 1.908538818359375, 0.6626831889152527, 0.34764090180397034, 0.8536459803581238, 0.19129221141338348, -0.7171699404716492, -0.25431421399116516, 1.0202215909957886, -0.448866069316864, 0.2982259690761566, -0.3718266487121582, -1.8924756050109863, -2.0406246185302734, -0.97712641954422, 2.123586654663086, 0.2811380922794342, 0.0027759193908423185, 1.984999179840088, -1.2036734819412231, -0.4440793991088867, -2.219276189804077, 0.05530792847275734, 1.6878668069839478, 1.0355441570281982, 0.8978878259658813, -0.2179214060306549, 0.7149181365966797, -0.3286481201648712, 0.6228618025779724, 0.8879694938659668, -0.2739461362361908, -1.2511330842971802, 0.06114673241972923, 0.6347928047180176, -0.011349744163453579, 0.5148795247077942, 0.4766812324523926, -0.8502774238586426, -0.6252401471138, 0.7300100922584534, -0.21965152025222778, -1.451952338218689, -0.759715735912323, -0.41724252700805664, -0.06846636533737183, -0.8865557909011841, 0.584404706954956, -0.7688890099525452, 0.7035801410675049, 0.46933338046073914, 1.3341529369354248, -1.6780720949172974, -1.6347771883010864, -0.5063655376434326, 0.7320795059204102, 0.6830968856811523, 0.06852476298809052, -0.20774662494659424, 0.6809715628623962, 0.04286617785692215, 1.0425291061401367, -0.4502226412296295, -1.228621482849121, -1.3800214529037476, -2.398887872695923, 0.2889138460159302, 1.1733523607254028, 0.33469507098197937, -0.7068405151367188, 0.8772168755531311, -0.4240046441555023, 0.6292430758476257, 0.7019757628440857, 0.3710070848464966, -0.059223685413599014, -0.4799805283546448, 0.048068128526210785, -0.5560479164123535, -1.337770700454712, -1.6653069257736206, 1.0192605257034302, -0.053197816014289856, -0.44974103569984436, 1.9274017810821533, -1.8362213373184204, -0.12702089548110962, 0.2687281668186188, -2.1260759830474854, 0.6091143488883972, -0.15895342826843262, -0.6830570697784424, -0.4690488576889038, 0.1430724859237671, -0.26997271180152893, 0.13071459531784058, 1.2462400197982788, 0.9315361380577087, 1.191899299621582, -0.34094178676605225, -1.0642602443695068, 0.943845272064209, -0.8959930539131165, 1.5493190288543701, -1.0799243450164795, -0.8813389539718628, -0.8130353689193726, -0.4848441183567047, 0.6173681020736694, -0.8848934769630432, -0.26923200488090515, 0.4936677813529968, -0.30707138776779175, 0.12321216613054276, 0.926571249961853, -0.07125767320394516, -0.40158313512802124, -0.45185917615890503, 1.091386079788208, -2.4860310554504395, -0.19270238280296326, 0.5098394155502319, -0.20004898309707642, 0.10918224602937698, 0.65123450756073, 1.2939796447753906, -1.2237529754638672], [1.0402699708938599, 0.4111092686653137, -0.33759450912475586, 1.593855619430542, 0.5325272083282471, -0.7173963785171509, -0.06226300820708275, 1.6090295314788818, -0.17014773190021515, 1.4112175703048706, -0.6587958931922913, -0.09612123668193817, 0.8225095868110657, -1.080165147781372, -1.0146713256835938, -0.5958673357963562, -1.8953815698623657, 0.4014565646648407, -0.37960514426231384, -0.6787364482879639, -0.17748811841011047, -0.12818898260593414, 1.8339176177978516, 1.3642688989639282, 0.5904527306556702, -1.7142378091812134, 0.03568247705698013, -0.6692461371421814, -1.8561960458755493, 0.11261063069105148, -0.4084793031215668, 0.06481032073497772, 0.7464799880981445, -1.5583817958831787, 0.5167580246925354, -1.0038915872573853, -0.8658377528190613, -0.13159005343914032, -0.6893579959869385, -1.2360882759094238, -1.0802143812179565, -0.23472154140472412, 0.142645001411438, -0.10397610813379288, -0.6720511317253113, 0.14353375136852264, 0.986396074295044, -0.07223127037286758, -1.8041988611221313, -0.3997587561607361, -2.1499204635620117, 0.33813774585723877, -0.41477909684181213, 0.08696845918893814, -0.8701530694961548, 0.35688549280166626, 0.6430008411407471, -0.28247106075286865, 0.8582457900047302, -1.4024834632873535, -1.14570951461792, 0.020015191286802292, 0.4372633099555969, 0.6028147339820862, 1.2387278079986572, 0.4357275366783142, -1.1668485403060913, 0.5316970348358154, 1.7289148569107056, 1.0409045219421387, -1.0423616170883179, -0.6211026310920715, 1.0860258340835571, -0.08637315779924393, -1.017724871635437, 1.166495680809021, 0.9700106382369995, 1.4722144603729248, -0.6136760115623474, -1.0299996137619019, -1.0962496995925903, -2.801703691482544, -0.10000910609960556, 0.4632323682308197, -1.0364536046981812, 2.424311876296997, 1.5240931510925293, -0.08351239562034607, -0.048172831535339355, -1.3616135120391846, -0.8241420388221741, 0.2945998013019562, -0.33163028955459595, -0.45987626910209656, 2.7676641941070557, -0.6932682394981384, 0.434856116771698, -0.20806531608104706, 0.4252488911151886, 0.4132102131843567, -1.630212664604187, -0.421275794506073, -0.7270743250846863, -2.2382938861846924, 0.037303801625967026, 1.654112458229065, -1.638506531715393, 1.7913081645965576, 1.5087991952896118, 0.059944599866867065, -0.24590642750263214, 0.11004901677370071, 0.0033030116464942694, 0.33856087923049927, -0.7113391160964966, -0.691559374332428, -0.06276581436395645, -1.0863178968429565, 0.43133461475372314, 1.0349838733673096, 0.9224427938461304, -1.3303147554397583, -1.1357089281082153, 0.09524770081043243, 1.0753124952316284, -1.7664459943771362, 1.0329594612121582, -0.1643710434436798], [-0.46866899728775024, -1.3744637966156006, -0.9535083770751953, -0.2785377502441406, 0.9137483239173889, -1.0580470561981201, -0.39528888463974, -0.2548530399799347, -0.36987966299057007, -0.2920916676521301, 0.25259512662887573, 1.385570764541626, 0.7173686027526855, -0.21979203820228577, -0.7317805886268616, 1.7980531454086304, -0.8974564075469971, -0.25431737303733826, 0.5098240971565247, 0.9941586256027222, -0.034071020781993866, -0.6754676699638367, 0.1518663913011551, -0.13505496084690094, 0.1839912235736847, 0.1775454878807068, 1.024337649345398, -0.3680254817008972, -0.24464355409145355, 0.11177858710289001, -0.16990327835083008, -0.1556742936372757, -1.3167953491210938, -1.3043984174728394, 0.7302943468093872, 0.6693836450576782, 0.5160879492759705, -1.7866990566253662, 1.756373643875122, -0.31097179651260376, 0.004696285352110863, 0.8501156568527222, -0.1393120437860489, -0.30172935128211975, -0.5387639999389648, -0.4270069897174835, 1.4074382781982422, 0.752261221408844, -0.45582717657089233, -1.4013969898223877, 1.4420528411865234, -0.28897151350975037, -0.07506993412971497, 0.9691882729530334, 0.43838363885879517, -0.3519471287727356, 0.8305183053016663, -0.7502581477165222, -1.3326526880264282, -0.953635036945343, -0.8835303783416748, 0.20280657708644867, -0.37801694869995117, -0.16291919350624084, -0.03176715970039368, 0.7628046274185181, 0.17684189975261688, 0.810081422328949, -1.0248035192489624, 0.8744313716888428, -2.0600104331970215, -0.3192649781703949, 0.5390130281448364, -1.2541886568069458, -1.3268555402755737, -0.07662903517484665, 0.6743925213813782, 0.6222792863845825, -1.9546031951904297, 1.0150151252746582, -0.8077126145362854, -0.6309487223625183, 0.6142206788063049, 1.0942192077636719, -1.2577650547027588, 0.34398728609085083, 1.4851007461547852, 0.9553831219673157, -0.16687022149562836, 0.3422626852989197, 0.30794090032577515, 0.5401721000671387, -1.058807373046875, -0.8566505908966064, 0.7569523453712463, -0.80257248878479, -0.27042487263679504, 0.0856296494603157, 1.0225319862365723, 0.3815861642360687, -1.0703798532485962, 0.09573042392730713, -0.9499753713607788, 0.6005348563194275, 0.16279897093772888, 0.14378400146961212, 0.3994791507720947, 0.9069555401802063, -1.1978323459625244, 1.7742136716842651, -0.42463648319244385, 0.14210018515586853, -0.8336694836616516, -0.09249050915241241, -0.6356672048568726, 0.8754464387893677, -1.587032437324524, -0.20149806141853333, 0.537361204624176, -0.42336440086364746, -1.0284384489059448, 0.879483163356781, -1.9533945322036743, 1.0426251888275146, 2.478853225708008, 0.5618991255760193, -0.7815229296684265, -0.27158644795417786], [-0.19484132528305054, 0.7976270318031311, -0.9789072871208191, -1.1290442943572998, -1.0609391927719116, -0.23961813747882843, 0.35236644744873047, -0.06730887293815613, 0.7111486792564392, -0.2728062868118286, 1.1074273586273193, -0.3754342496395111, 1.371081829071045, -0.615337073802948, 0.3590526878833771, 0.8979552984237671, 0.5122601985931396, -0.4915163516998291, -0.29228973388671875, -0.7483097910881042, -1.206057071685791, -0.6307675242424011, 1.9867295026779175, 1.3802399635314941, -0.02926473878324032, -0.7651569247245789, -0.6433616280555725, -0.8552622199058533, -0.24891671538352966, 0.5542259216308594, 0.8216432332992554, -0.0548778735101223, -0.832649290561676, 0.6340852379798889, -0.36117225885391235, 0.07579293102025986, -1.1315463781356812, 0.721422553062439, 0.8085997104644775, 0.7643422484397888, -1.3107924461364746, -0.33408045768737793, -0.1505734920501709, -1.0719133615493774, 0.42514851689338684, 1.8821117877960205, -0.35565581917762756, -1.3940629959106445, -0.2503070533275604, 0.007154497783631086, 0.2465711385011673, 1.4162216186523438, 0.03581209480762482, -1.3765745162963867, -1.2839993238449097, -0.08331464231014252, 0.6076202988624573, 2.1931443214416504, -0.2355446070432663, -0.5418788194656372, 1.3501436710357666, 1.2215919494628906, 1.48508882522583, 0.6004460453987122, 0.5210123658180237, -0.7084271907806396, -1.2375661134719849, 1.0370193719863892, -0.050957683473825455, -1.6014207601547241, -0.00786173902451992, -1.0498932600021362, -1.6936230659484863, -0.3251723349094391, -2.3684909343719482, -0.8002482056617737, -1.6178346872329712, -0.13655619323253632, -0.7748346924781799, 0.20695796608924866, 1.0278596878051758, 1.2657296657562256, 1.0979435443878174, -1.0778775215148926, -0.30560198426246643, -1.3120629787445068, 1.4116885662078857, -0.7647525072097778, 0.4034949541091919, -0.6643055081367493, -0.13683545589447021, 0.08163817226886749, 0.19103200733661652, 0.9684785008430481, -1.4915111064910889, -0.30101022124290466, -0.19378454983234406, -0.244557186961174, -0.060222700238227844, 0.6234971880912781, 0.12467313557863235, -0.9989047646522522, 1.2094701528549194, 1.561117172241211, -0.7056915163993835, -0.475753515958786, -1.3665547370910645, -0.7084056735038757, 0.41860589385032654, -0.11744586378335953, -0.8388177752494812, -0.22112345695495605, 0.19571128487586975, -1.062274694442749, 0.5426801443099976, -1.079958200454712, -0.9868859648704529, -0.4572259485721588, 0.877217173576355, -1.11020827293396, -0.5579906105995178, -0.15018384158611298, -0.5580126643180847, -0.8368929624557495, 0.33665233850479126, 1.1634836196899414, -0.9789944291114807, -0.38430190086364746], [-1.946847677230835, -0.13555823266506195, 1.1761475801467896, 0.8307536244392395, -1.3438127040863037, 0.9148232936859131, 0.23933030664920807, -0.9152349829673767, 0.0983501598238945, 0.45380136370658875, -0.37419506907463074, -1.414035439491272, -0.2500946521759033, 0.482879638671875, -0.6372497081756592, 0.5449726581573486, -0.7196323275566101, -0.12848077714443207, 0.5198698043823242, -0.4920229911804199, 0.5013850331306458, -1.9122915267944336, 1.2301989793777466, 1.7312930822372437, -0.4840821623802185, 0.6332656145095825, 0.5538939833641052, -0.20586198568344116, -0.33828186988830566, -1.2391380071640015, 0.03371570631861687, 1.9298207759857178, 0.5375020503997803, 0.9973530173301697, -0.3966730535030365, -0.16624560952186584, -0.010502866469323635, -0.44148576259613037, -1.1907130479812622, 1.0598710775375366, -0.07746891677379608, 0.6568726301193237, 0.31020963191986084, 1.613728404045105, 1.8979460000991821, -2.0515248775482178, -1.300172209739685, -0.8573914170265198, -0.8079600930213928, -1.4209715127944946, 0.7182481288909912, 0.3800041973590851, -1.167759656906128, -0.24729560315608978, 1.3366774320602417, 0.9689831137657166, 0.19563952088356018, -2.0930423736572266, 1.649009108543396, -0.5993590950965881, -0.4636227488517761, -1.4482313394546509, 0.634361743927002, -1.3403383493423462, 1.3681886196136475, -0.6434767842292786, 1.0855408906936646, -1.7556604146957397, 0.5282528400421143, -0.39603447914123535, -1.4417345523834229, 0.6476134657859802, -0.1779734045267105, -0.7444612979888916, 0.5746996402740479, -0.7478896975517273, -0.2685471475124359, 0.5741972327232361, -0.01322050578892231, 0.0671086236834526, -0.5292473435401917, 0.05232886224985123, -0.5621556043624878, 0.30552706122398376, -0.9834911227226257, 0.5592495799064636, 0.6537949442863464, 2.1551201343536377, 2.028184413909912, -0.288700670003891, 0.6167168021202087, 3.1569719314575195, 0.4911591708660126, 1.4564788341522217, 0.4781476557254791, 2.0158424377441406, -1.15622878074646, -1.1458806991577148, -0.7713309526443481, -0.4446820616722107, -1.0752288103103638, -0.5505731105804443, 0.08824137598276138, -1.3822146654129028, 0.467741996049881, 1.6809059381484985, -2.671309232711792, -0.1193493902683258, -0.7762852907180786, 1.5985468626022339, -2.6097986698150635, 1.0525562763214111, 1.6876368522644043, -0.1588064432144165, 1.2771131992340088, -0.496687650680542, 3.021300792694092, 0.14809051156044006, -0.29489046335220337, -1.1442173719406128, -1.1932748556137085, 0.6483956575393677, 0.3740265965461731, -0.712079644203186, 0.846028208732605, 0.4932765066623688, -0.4772796928882599, 0.20261482894420624], [0.2508567273616791, -0.23993350565433502, 0.42272353172302246, -0.9842902421951294, -0.5577648878097534, -0.2941223680973053, -0.7366825342178345, 0.1920536905527115, -1.9221194982528687, -0.28000354766845703, -1.09846031665802, 0.5015076398849487, 1.255519986152649, 1.096904993057251, 0.931865930557251, -0.7773432731628418, 2.4222500324249268, 1.0312693119049072, 0.3125624358654022, 1.585231900215149, -0.06427749246358871, -0.20251259207725525, -0.5874168276786804, -0.6247450113296509, -0.4776628315448761, 0.17664094269275665, 1.9957534074783325, 0.22881850600242615, -2.268317699432373, 0.49643048644065857, -1.0439070463180542, -0.06813254207372665, 1.0840367078781128, 1.2362741231918335, -1.187960147857666, 1.3974409103393555, 0.8727299571037292, -1.0234870910644531, 0.7355571389198303, 1.139582872390747, 0.3487083613872528, 0.9617677927017212, -0.22979441285133362, -0.8833169341087341, -0.47276952862739563, 0.9430665969848633, 0.5835186839103699, -1.9945217370986938, 0.5335757732391357, -0.36393049359321594, 0.42105090618133545, -0.04740944132208824, 0.14804252982139587, -0.7028600573539734, 1.1684911251068115, 0.9926083087921143, -0.5434820652008057, -0.4131089746952057, -0.47351789474487305, -0.3173872232437134, 0.35202059149742126, 0.8360057473182678, 0.331195205450058, -2.589999198913574, -0.3137657940387726, -0.10949191451072693, 1.1697534322738647, -0.1711028516292572, 0.5640501976013184, -0.8992529511451721, -0.09170712530612946, 0.49018150568008423, -0.30734848976135254, -0.4353352189064026, 0.6011432409286499, 0.44892922043800354, 1.4706923961639404, 0.4145493507385254, -0.0835910215973854, 2.1267969608306885, -0.9739497303962708, 1.7875534296035767, 0.649452269077301, -0.43701279163360596, -0.027280984446406364, 0.7541736364364624, 0.995394229888916, -0.26015642285346985, -0.3385592997074127, 0.7380695939064026, 0.6437409520149231, 0.3093648850917816, 1.0577661991119385, 0.38622158765792847, 0.7778230309486389, 0.9839024543762207, -0.8103386759757996, -1.1059415340423584, -0.3658018410205841, 1.1051818132400513, 0.08599726110696793, -0.05344926565885544, 1.6824250221252441, -0.668764054775238, 1.1382911205291748, 0.6969306468963623, 0.2980101406574249, 0.37052762508392334, -0.6131600141525269, 2.030001163482666, -1.596899151802063, -0.9050925374031067, 0.08427165448665619, 1.4749480485916138, 0.6657230257987976, -0.06223833188414574, -0.599493682384491, -0.759333074092865, -2.219998836517334, 0.21690529584884644, 0.34957820177078247, -2.2117090225219727, 0.45739150047302246, -0.6418132185935974, 0.500273585319519, -0.9532528519630432, -0.9794701933860779, -0.5301478505134583], [-0.30302682518959045, 0.7447476387023926, 0.18144004046916962, 1.048503041267395, -1.5848506689071655, 0.47211530804634094, 0.9753791093826294, -0.5196091532707214, 0.35344138741493225, -0.6721628308296204, -1.498076319694519, -1.0012953281402588, -1.443620204925537, 2.0562164783477783, 0.28015294671058655, -0.09098245203495026, 1.3036280870437622, -1.0005244016647339, 1.204768180847168, 0.2721237242221832, -0.04187673702836037, -1.3793424367904663, -0.7387359738349915, -0.7666682004928589, 1.3524385690689087, 1.4398709535598755, 0.09323042631149292, -0.4574335217475891, -1.178382158279419, 0.7117549777030945, 0.24051141738891602, -1.809816598892212, 0.10988466441631317, -0.07332787662744522, -1.721738338470459, 1.7297031879425049, -0.4651215374469757, 0.34182077646255493, 0.07693693041801453, 0.06905852258205414, 0.40703216195106506, 0.3663720488548279, -0.10432074218988419, -0.558548629283905, -0.6019629836082458, -0.9990251660346985, 0.4390985369682312, -1.0407134294509888, 0.2716100513935089, -0.3228283226490021, -0.6492611765861511, -0.8904920816421509, -1.576295256614685, -0.7007920742034912, 0.9034233093261719, -0.011900363489985466, 0.6279261708259583, -0.09684097021818161, -2.060084581375122, -0.8239456415176392, -1.781545877456665, -0.8663676977157593, 0.09261907637119293, -0.9205144643783569, 2.5762922763824463, 0.24886460602283478, -0.60568767786026, 1.0732723474502563, 0.801115870475769, -0.26058724522590637, 1.2931290864944458, -0.5839712619781494, 1.3819546699523926, -0.41340944170951843, -0.992882251739502, 0.310590535402298, 0.525597095489502, 0.9620785713195801, 0.35475459694862366, 1.206913709640503, -0.7662892937660217, 0.46162867546081543, 0.7739949822425842, -0.12552011013031006, 0.12490461021661758, -0.25292539596557617, 0.15888218581676483, 0.6873282194137573, 0.3790193498134613, 0.2689201831817627, -0.7086460590362549, -0.16439104080200195, 0.7301995158195496, -0.7078596353530884, 0.6455357670783997, 0.17169331014156342, -0.7100911140441895, -0.6439732909202576, 0.03602290153503418, -0.8631600737571716, -0.6739940643310547, 1.2551261186599731, -0.03382064029574394, 2.2359061241149902, 0.5577249526977539, -1.016548991203308, 0.5046797394752502, -1.9606150388717651, 0.39919760823249817, 1.1690598726272583, 0.673641562461853, -0.12884953618049622, -1.6274608373641968, 1.032865285873413, 0.8996375799179077, -0.07653399556875229, 0.9693794250488281, -0.1413610726594925, 0.5764407515525818, -0.35340404510498047, 0.08200060576200485, 0.01428989227861166, -1.1614100933074951, 1.484642744064331, -0.557924747467041, -0.3951444625854492, 0.8807836174964905, -0.33745911717414856], [-0.36728060245513916, -1.700668454170227, 0.8070375323295593, -1.641953706741333, 0.11096663028001785, 0.464455783367157, -0.11723531782627106, 0.10718569904565811, -0.1462050974369049, -1.0621637105941772, -0.6313258409500122, 1.2196944952011108, -1.1647367477416992, -0.5858350396156311, 2.354156017303467, 0.0726846307516098, 0.34512627124786377, -1.3784698247909546, 0.4683600068092346, 0.4833366274833679, -0.21232318878173828, 1.6824402809143066, 0.9182522296905518, 0.3482670187950134, -0.8588994145393372, -0.004591784905642271, -0.5938829779624939, 0.11276789009571075, -0.0624571293592453, -1.744452953338623, 0.27553749084472656, -1.0220179557800293, 1.0237761735916138, 0.5814929008483887, -0.6238046288490295, -0.3524368703365326, 0.30806484818458557, -0.18928395211696625, 0.7782419919967651, -2.12151837348938, 1.3385257720947266, -0.67791748046875, -0.9305326342582703, -0.35139966011047363, 1.0200345516204834, 1.0918117761611938, -0.5460827350616455, -0.5668724179267883, 0.9155851006507874, 2.202388286590576, 1.7152442932128906, 1.0639640092849731, 0.23581670224666595, -1.1173114776611328, 0.8513153791427612, 1.36105215549469, -0.1580703854560852, -0.32684144377708435, 0.6645218133926392, -0.5948538780212402, 0.3518357276916504, -0.2027769535779953, 1.4683969020843506, 1.9535115957260132, -0.4593350887298584, -0.3539980351924896, 0.3226657509803772, 0.4466375708580017, 1.686556339263916, 0.19439566135406494, -0.814921498298645, -0.22939378023147583, 0.09874451905488968, 0.220535546541214, 1.406864881515503, 0.44578033685684204, -0.0028718304820358753, 1.8626614809036255, 2.0855069160461426, 0.1707308292388916, -0.8214120864868164, 1.580119013786316, -0.008022062480449677, -1.5942256450653076, -0.08043447881937027, 0.8170623779296875, -1.0057237148284912, 0.7341718077659607, -0.27692079544067383, 0.5784449577331543, 2.088522434234619, -0.47763770818710327, 0.6632776260375977, -0.25228092074394226, 0.6477422118186951, 1.924519658088684, 1.145922303199768, 0.18849259614944458, 0.8106942176818848, -0.43536603450775146, -0.9378581643104553, 1.0929843187332153, 0.5283737778663635, 1.9517431259155273, -0.35890933871269226, 0.4748435318470001, -0.7932918071746826, -0.4388607442378998, -1.084252953529358, 1.6676304340362549, 1.2238147258758545, 0.17422321438789368, 0.043755173683166504, -0.7017186880111694, -1.2961047887802124, -1.0222238302230835, -1.066505789756775, -0.37212297320365906, 0.27635815739631653, 0.16225413978099823, -0.08138316869735718, -1.2738103866577148, -1.3811028003692627, -2.7646312713623047, -0.14230282604694366, 0.42461511492729187, 0.2247702032327652, 2.034637928009033], [-1.0836443901062012, -0.7573887705802917, 2.23093581199646, -0.08368716388940811, 0.01066607516258955, -0.467582643032074, -1.875239372253418, 0.1757371872663498, 0.2042006403207779, 0.9192975163459778, 0.6742594242095947, -1.880578637123108, -0.6833991408348083, -1.0020804405212402, 1.626721978187561, -0.633516788482666, 0.029771216213703156, 0.03724227100610733, 0.6809834241867065, -0.4442664086818695, -0.7554796934127808, 1.007670283317566, -1.6740953922271729, -0.673401415348053, -0.06042635440826416, 1.28988778591156, 0.9110578894615173, 0.4260658621788025, 0.2355007380247116, -1.0915240049362183, 0.9120579361915588, 0.16800497472286224, -0.3341251015663147, -0.03355412557721138, 0.3294014632701874, 0.3022836148738861, -0.36600542068481445, -0.08779776096343994, 3.6343178749084473, 0.5816145539283752, -1.2798938751220703, 0.8433367609977722, 2.2120981216430664, -0.12018678337335587, 0.7269144654273987, 0.49549293518066406, 0.9328096508979797, -1.0032763481140137, 0.748897910118103, 0.8577982783317566, 0.9386937618255615, -0.07262461632490158, -0.2273472100496292, -0.7315860986709595, 0.4257935881614685, 1.6053705215454102, -1.2227258682250977, -0.790221631526947, -0.3500225841999054, -0.1836978793144226, 0.22738374769687653, -2.415100574493408, -0.911518394947052, 0.16577234864234924, -0.699110209941864, -0.6919373869895935, -1.1250046491622925, -0.7072740793228149, 0.6686814427375793, -0.7780935764312744, 1.1918137073516846, -0.47134873270988464, -0.42114341259002686, -0.13949868083000183, 1.2272696495056152, 0.42698317766189575, 1.5914993286132812, -2.01488995552063, 0.8539997935295105, 1.5260772705078125, -0.8463367223739624, -1.5742589235305786, 0.5737765431404114, -1.3313634395599365, -0.7683023810386658, -1.39936363697052, -2.1210384368896484, 0.505627453327179, 0.6707954406738281, 0.32122084498405457, 1.3739573955535889, -0.25790950655937195, -0.6618574857711792, 2.0906991958618164, 0.5171453356742859, -0.854315996170044, -0.8970180153846741, 0.5361940860748291, -0.6539052128791809, -1.097030758857727, -0.9813016057014465, 0.9515073299407959, 0.15422731637954712, -2.2137551307678223, -0.9982426762580872, 0.17877110838890076, -0.6249642372131348, 1.1882407665252686, -0.5067428350448608, 0.709804117679596, 0.19396963715553284, 0.48940691351890564, 1.302432894706726, 1.714707612991333, -0.2296311855316162, 1.8498666286468506, -0.03295181691646576, 1.2536699771881104, 0.29280275106430054, 0.46158167719841003, 0.7784146666526794, 2.0832645893096924, -0.7639017701148987, 0.7633717656135559, -0.6316527724266052, 0.45881223678588867, 1.5510870218276978, -0.24369242787361145], [-0.31459328532218933, -2.0367636680603027, -0.33599650859832764, -1.0412061214447021, -1.8263838291168213, -0.5287734866142273, 1.0488630533218384, 1.8230922222137451, 0.9587706327438354, 0.9069522023200989, 0.42755332589149475, -0.38193899393081665, -0.921232283115387, -3.1388027667999268, -0.2965559661388397, 0.9019576907157898, -0.9181884527206421, 1.2776440382003784, -0.7445796728134155, 1.1332626342773438, -0.3382485806941986, -0.9491162300109863, -0.7258099317550659, -0.3921669125556946, 2.314566135406494, 0.016890501603484154, 0.0382838137447834, 0.7579405903816223, -0.7770668864250183, 0.5001147985458374, 0.12072101980447769, 0.47516632080078125, -1.7499992847442627, 0.8990859389305115, -0.05872562900185585, 0.6612188816070557, 1.4463372230529785, 0.6005052924156189, -0.28823593258857727, -0.7547862529754639, 1.2505508661270142, 1.120345950126648, -0.0055048177018761635, 0.20546361804008484, 0.3705083429813385, 1.3261092901229858, -0.15738876163959503, 1.5165473222732544, 0.13898006081581116, -0.7393558621406555, 1.3281203508377075, -0.07083253562450409, 0.1884646713733673, 0.018805982545018196, -0.1530657708644867, -0.9787166118621826, 0.042124830186367035, -0.9088263511657715, 0.9684000611305237, 0.6757180690765381, -0.2008449137210846, -0.5388196110725403, 1.3199435472488403, -0.8520290851593018, -1.7856695652008057, 1.1413593292236328, -1.9827322959899902, -0.8055394291877747, 1.994470238685608, -1.6903362274169922, 0.42734697461128235, -0.96957927942276, -2.2834620475769043, -0.4667263329029083, 0.4864130914211273, 0.10145718604326248, 0.48646923899650574, -1.606905460357666, 1.5674813985824585, -1.7199232578277588, 0.7359611392021179, -0.4490409791469574, -1.6480942964553833, 0.41202211380004883, 0.027708081528544426, 0.8045875430107117, 0.15235306322574615, -0.21018511056900024, 0.14955662190914154, 1.1686893701553345, -0.7318804860115051, -0.9859628677368164, 0.2942966818809509, 1.004874348640442, -0.7302395701408386, -1.6380640268325806, 1.7381436824798584, 0.9016152620315552, 0.2947673797607422, -2.4246456623077393, 1.3045048713684082, 0.7689491510391235, 1.51067054271698, -1.5131235122680664, -0.7501790523529053, -1.5320370197296143, -0.7466010451316833, -1.2075494527816772, -0.2013295292854309, -1.0048872232437134, 1.9221229553222656, 0.1130409762263298, -0.08655967563390732, 0.6431892514228821, 1.0575120449066162, 0.8276830315589905, 1.8113311529159546, 1.1142468452453613, 1.2121680974960327, -0.6498532891273499, -0.5488755106925964, -0.9844439029693604, 0.13836030662059784, 1.4194649457931519, 0.894100546836853, -0.5632196664810181, 0.3521643579006195, -0.8433828353881836], [-0.487956166267395, -0.9528456926345825, 0.5463895797729492, -1.5365509986877441, -1.7607978582382202, -0.0509113073348999, -0.9489482641220093, -0.2344321459531784, -0.04799247905611992, 2.076046943664551, -1.2362364530563354, 0.04217638447880745, 1.2453798055648804, 0.1964419037103653, -0.07354826480150223, 0.8968124389648438, 0.04607381671667099, -0.5488448143005371, 0.7203897833824158, 1.3329010009765625, -0.16599437594413757, -1.5731967687606812, -0.5817790031433105, 1.9174997806549072, 0.9463617205619812, -0.5991880893707275, -0.019233204424381256, -1.754293441772461, -0.1827675849199295, 0.7515720725059509, 1.0166683197021484, -0.2396925985813141, -0.9620257616043091, 0.4056514799594879, -1.7035481929779053, -1.0705379247665405, -1.1687744855880737, -0.868891716003418, -0.6826076507568359, 0.244212806224823, 1.0855917930603027, -0.9193931221961975, -0.17109344899654388, 2.062236785888672, -0.6052876710891724, -1.8359394073486328, -0.22931957244873047, -0.43718764185905457, -0.814658522605896, 0.8961652517318726, 0.7522242069244385, -0.45242899656295776, 0.7118297815322876, 1.19295334815979, 1.019230842590332, 0.5137546062469482, 0.6410251259803772, -1.2376359701156616, 0.7275769114494324, -0.7363953590393066, -2.2519400119781494, 0.837110161781311, 0.7803142666816711, 1.1105092763900757, -0.20413871109485626, -1.8026782274246216, 1.1665173768997192, 2.0269994735717773, -1.7309352159500122, -1.0130949020385742, -0.9446208477020264, 0.4906140863895416, 0.01985669881105423, 0.3949427008628845, 0.6291666626930237, -2.324821949005127, -0.7913597822189331, 0.6603208184242249, 0.20526206493377686, -0.32032689452171326, 0.541037380695343, 0.4677240252494812, 0.3123227655887604, -0.5884786248207092, 0.865424394607544, 0.37311050295829773, -0.10500217974185944, -0.29135268926620483, 0.875413715839386, -0.36227041482925415, 0.5452023148536682, 2.5225327014923096, -0.5688267350196838, -0.38357946276664734, 0.11751504987478256, 0.21114735305309296, 0.45322489738464355, 0.5376341342926025, 0.5848048329353333, 0.37889593839645386, -0.19360636174678802, 0.42375218868255615, -1.6232123374938965, -0.3261396288871765, -0.22748929262161255, 0.06888879835605621, 0.49886438250541687, 1.1753357648849487, 0.1755550652742386, -1.6923233270645142, -0.8772344589233398, 0.21194201707839966, 1.4606767892837524, 0.0028054192662239075, -0.2953278124332428, 0.47593221068382263, -0.669317364692688, -0.4987484812736511, -2.033099412918091, -1.1831979751586914, 0.5940155386924744, 0.20081353187561035, 1.8990908861160278, -0.5107250213623047, 1.4406236410140991, 0.3277500569820404, 0.7745837569236755, -0.6284968852996826], [0.21479374170303345, -0.40620118379592896, -0.8212169408798218, 1.2261375188827515, 0.7110751867294312, -0.693163275718689, -0.30803942680358887, -0.9085435271263123, -1.101157546043396, 1.1418406963348389, -0.4048633873462677, -0.25526729226112366, 0.23081794381141663, -1.1026124954223633, 0.6772746443748474, 0.7352557182312012, 1.1579657793045044, -0.3449596166610718, 1.0752590894699097, 0.6989462971687317, 1.8257746696472168, -0.11658232659101486, 1.2832131385803223, 2.130291223526001, 1.2637310028076172, 1.2040514945983887, 0.5011571645736694, 0.009234068915247917, -0.44568103551864624, -0.08495225012302399, -0.43304747343063354, -2.0702176094055176, 0.20514264702796936, 0.9038992524147034, 0.10800683498382568, 0.018398789688944817, 0.07434280961751938, 0.8581507205963135, 0.4864928424358368, 0.7542284727096558, 0.4282507002353668, 0.05358007177710533, 0.5984293818473816, -1.2483186721801758, 0.17586266994476318, -0.023505065590143204, 1.9069225788116455, 0.9832232594490051, -0.15781140327453613, -0.26605620980262756, 0.5120494961738586, 0.9355447888374329, 0.7960014939308167, -0.6368319392204285, 0.29247963428497314, 0.3408529758453369, 0.20101678371429443, 0.9763950705528259, -1.4003100395202637, 0.9260186553001404, 1.700813889503479, 0.3218652307987213, 1.2303745746612549, -1.8213250637054443, 0.37056422233581543, -0.5808091163635254, 1.576799750328064, -0.11329673230648041, 0.42052021622657776, -0.981731116771698, 1.112817645072937, 1.5286576747894287, 0.8547006845474243, 0.49022188782691956, 0.5928328633308411, -0.13327611982822418, 0.2815314829349518, -0.11675567924976349, 1.0900397300720215, 0.4643556773662567, -2.121703863143921, 0.917616069316864, 0.2712297737598419, 0.002573093632236123, -0.2570227086544037, 0.19145767390727997, -0.8282448649406433, 0.5138822197914124, -0.8987242579460144, -0.6229172945022583, 0.3573128581047058, 0.7815741300582886, 0.13370181620121002, -0.7935146689414978, -1.6991759538650513, -0.43374553322792053, 0.8021165132522583, -2.0589258670806885, 0.7431784272193909, -0.8303777575492859, 0.6859167218208313, -0.5800443291664124, -0.4309479296207428, -1.5575275421142578, 1.0105712413787842, -0.4445532262325287, 0.9737014174461365, -0.692768931388855, -0.20826612412929535, 0.5386514663696289, 2.587334632873535, 0.48074617981910706, 0.06400634348392487, -0.07080697268247604, 1.0337568521499634, -1.269263505935669, -2.1497814655303955, -0.9929943680763245, 1.3209911584854126, -0.6861065030097961, 1.4027100801467896, -0.6272496581077576, 1.5527681112289429, 0.8170902132987976, 0.2622532546520233, 1.1645565032958984, 1.0714678764343262, 0.8683826923370361], [0.9556649923324585, -0.04391658306121826, -0.4173509180545807, -0.6522276997566223, 0.16337214410305023, -1.1424616575241089, -0.4230096936225891, 0.9123860597610474, -0.8931049704551697, -0.6006821990013123, -0.2615429759025574, 1.55239737033844, -1.8039900064468384, -0.8377184867858887, -0.7696930170059204, 1.0249749422073364, 0.18801844120025635, -0.766228437423706, 2.7666256427764893, -0.4324679374694824, -0.49954351782798767, 0.8734753727912903, -1.007461667060852, 1.636337399482727, -0.22490419447422028, -0.6505895256996155, -0.4436732232570648, 0.7569763660430908, 1.7186940908432007, 0.4019233286380768, -1.3350862264633179, -1.6165391206741333, -1.4712390899658203, 2.1788365840911865, 0.5485052466392517, 0.14304617047309875, 1.0709774494171143, 1.7269366979599, -0.6124890446662903, 0.16590635478496552, -0.4740277826786041, -1.777622938156128, -0.795315682888031, 0.8533539772033691, 0.14624512195587158, -0.7325738668441772, -1.7642216682434082, -0.13895103335380554, 0.4834374785423279, -0.3396940529346466, -0.3092797100543976, 1.7839871644973755, -0.3712671101093292, 0.13946907222270966, -1.381393313407898, 2.145480155944824, 1.3611036539077759, -0.4637521207332611, 0.6967726945877075, 0.533798098564148, 0.01843511126935482, -0.5410475134849548, 2.1854166984558105, 0.3017579913139343, 0.0888894721865654, -0.4331018030643463, -0.07633557170629501, 1.5289628505706787, -1.698988437652588, -1.8645449876785278, -0.2037966549396515, -0.10939731448888779, -2.163771152496338, 1.3315118551254272, 0.1621176302433014, -1.042388916015625, -0.2048155516386032, 0.5357952117919922, 0.26127466559410095, -0.09787224233150482, 1.3634103536605835, -0.6355829238891602, -2.2847414016723633, 0.13738913834095, -0.23986051976680756, -0.6202138066291809, 0.13766515254974365, -0.9709419012069702, 0.3833056092262268, -0.7571335434913635, 0.9257636070251465, -1.0710580348968506, -0.01882063038647175, -0.8855639696121216, -0.7162607312202454, 0.17573225498199463, 2.1529390811920166, 0.28939810395240784, 1.3386527299880981, 0.12648046016693115, -0.7228606939315796, -0.9186837077140808, 0.4401589035987854, -1.9226778745651245, -1.0507493019104004, -2.6693332195281982, 2.117816925048828, -0.019326912239193916, -1.1452258825302124, -1.1045960187911987, -0.7843865156173706, 2.586501121520996, 1.0950847864151, -0.772487461566925, 0.6631252765655518, -0.1934179663658142, 0.2109318971633911, 0.09029021859169006, -0.12038371711969376, -1.3920233249664307, -0.8649480938911438, 0.4794329106807709, 0.8041195869445801, -1.1159526109695435, 0.7553095817565918, -1.4183032512664795, -0.10259480774402618, -2.086803436279297], [0.27167826890945435, 0.30296987295150757, 0.1900976598262787, 1.529467225074768, 0.6328993439674377, 0.34533897042274475, 0.08474551141262054, -1.1150895357131958, 1.1972118616104126, 1.077111840248108, 1.4526822566986084, -0.5081331729888916, 0.0051561687141656876, -0.46671053767204285, -0.8518803119659424, 0.9987821578979492, 1.346680998802185, 1.311681866645813, 0.14296138286590576, -0.6937257647514343, -1.1351863145828247, -0.24029308557510376, 0.13075187802314758, 0.1266704946756363, -0.5504571199417114, -0.583311140537262, -0.42911791801452637, 0.6193162798881531, 1.2869226932525635, -2.145571708679199, -0.44144701957702637, -0.2885880172252655, 0.438048779964447, -0.7867372035980225, 0.7269404530525208, 1.1216531991958618, 0.029149645939469337, 0.7334269881248474, 0.49614831805229187, -1.3549857139587402, -0.6382109522819519, -0.9048638939857483, -1.3193787336349487, 1.245666265487671, -0.8461575508117676, 0.6570461988449097, 0.340728223323822, -0.624776303768158, 1.3150180578231812, 0.16252867877483368, -0.6706214547157288, -1.3822789192199707, -0.734131932258606, 0.5699754357337952, 1.7543272972106934, 0.8877530694007874, -1.0524064302444458, -0.15189634263515472, 1.4459470510482788, -1.313200831413269, 0.7410101294517517, -1.079062819480896, 0.643395721912384, 0.5608199238777161, -1.0268042087554932, 0.14532101154327393, -0.7991641163825989, -1.587005615234375, 0.4669930040836334, 0.11657451838254929, 0.7787417769432068, -0.13905496895313263, -1.4311972856521606, 0.6893892288208008, 0.7810511589050293, 0.5996443033218384, -2.3721463680267334, 0.32231950759887695, -1.2004843950271606, 0.2495739609003067, 0.8514232039451599, 0.03464771807193756, -1.2686772346496582, -0.5038789510726929, -0.5175712704658508, -0.9969891309738159, 0.2108699381351471, 1.5933693647384644, 0.38888129591941833, -0.41067737340927124, -0.3434445858001709, 0.20011112093925476, 0.2127160131931305, 0.133962944149971, -0.6907526850700378, -0.12351852655410767, 0.5494455695152283, -0.5236321091651917, -0.4273117482662201, -0.8731732368469238, -1.095407247543335, -1.8187339305877686, 0.6770068407058716, -0.0003696621861308813, -1.8145464658737183, -0.7131937742233276, 0.19880925118923187, 0.766201376914978, -0.7718883752822876, -0.25268399715423584, -1.616958737373352, 0.5612823963165283, -0.6845535039901733, 2.2239723205566406, 1.6229476928710938, -0.302465558052063, 1.0342941284179688, 1.6607550382614136, -1.564324140548706, -0.94158935546875, 0.8946191072463989, -0.04874951019883156, -2.3956680297851562, -0.029706522822380066, -0.8309895396232605, 0.22530251741409302, -0.04334733635187149, 2.3668603897094727], [1.2914568185806274, -0.35496821999549866, -1.0024434328079224, -0.7916757464408875, 0.9329003095626831, -0.47998350858688354, 0.2821064591407776, -1.0067825317382812, 0.14661556482315063, -0.29833415150642395, 0.041945282369852066, -1.3963466882705688, 0.8882914185523987, 0.45859453082084656, 0.3150959312915802, 0.3331293761730194, 0.0032558580860495567, 1.7385376691818237, -0.09418335556983948, 0.6328367590904236, -1.411446452140808, 0.9723249673843384, -0.2108072191476822, 0.20680925250053406, -0.3223339021205902, -0.33656221628189087, -0.6846330761909485, -0.24006178975105286, -1.804364800453186, -0.20408880710601807, 1.9717732667922974, -0.2616361081600189, 0.34265878796577454, -0.09788691997528076, 0.13230152428150177, -0.8715699315071106, 0.34591144323349, 0.9347214698791504, 1.157841682434082, -0.822559118270874, -0.3067520260810852, 0.1554298996925354, -0.42871618270874023, -0.5621967315673828, -0.8154863119125366, -0.8855084180831909, -1.1259816884994507, -0.9551960825920105, -1.2114324569702148, -0.3166273236274719, -2.139953136444092, 0.29676780104637146, -0.6759907007217407, 0.5287061929702759, -1.6328140497207642, 0.4693102538585663, 0.8515399098396301, -0.6840108633041382, 0.6263846158981323, 0.12928703427314758, -0.8858048319816589, -0.1677556335926056, 0.2772899866104126, 1.964917540550232, -1.0189294815063477, 0.5090116262435913, -0.2459275871515274, -0.32612013816833496, -0.7469362616539001, -1.123869776725769, 0.09974849969148636, -1.5210367441177368, -0.6025691032409668, 0.029139846563339233, 2.280263662338257, 1.179084062576294, -0.1858517825603485, -0.3362036347389221, 0.08063830435276031, -2.3561999797821045, 0.7598434686660767, -0.03403971716761589, -0.8880711197853088, 0.9638166427612305, -0.11694132536649704, -0.9770529866218567, -1.4829905033111572, 0.6017250418663025, -1.8432410955429077, -0.8816326856613159, -0.13586720824241638, -0.006746856961399317, -0.37310901284217834, -0.3293705880641937, 1.8112919330596924, 0.527472198009491, 0.8624919652938843, -0.23495101928710938, 0.922637403011322, -0.24759972095489502, 0.5148532390594482, 0.14004741609096527, -0.8568695187568665, 0.9595872163772583, -2.0621981620788574, 0.4279019832611084, -0.3576062023639679, -0.9999728798866272, 1.1076295375823975, 1.9202104806900024, -0.9870116114616394, 1.5997707843780518, 0.670502781867981, -0.31186288595199585, 0.3604341149330139, -0.11859039217233658, 0.6341373920440674, -0.9806453585624695, 2.432573080062866, 0.30377060174942017, 1.7897965908050537, -0.8600634336471558, 1.7715699672698975, -0.4037119448184967, 0.5532687902450562, 0.45274800062179565, -0.15780793130397797, 0.07269364595413208], [-0.42144089937210083, 1.3409217596054077, -1.5696561336517334, 0.5682983994483948, -0.2079555243253708, 0.8923836350440979, -0.5317534804344177, -0.15489225089550018, 1.3771061897277832, 1.6214832067489624, 0.5514208674430847, -0.05371249467134476, -0.7280784249305725, -1.2305890321731567, 0.4108739495277405, 0.566504955291748, 0.5500465035438538, 2.204131841659546, 0.787712812423706, -0.010795067064464092, 2.5082032680511475, 0.9746017456054688, -1.2404295206069946, 0.9176639318466187, 0.21113380789756775, -0.12497640401124954, 0.6221578121185303, 1.4068548679351807, -0.39673084020614624, 2.4263086318969727, 0.35435256361961365, -0.8835663795471191, 1.5816757678985596, -1.6411951780319214, -0.5146673917770386, 0.18971872329711914, -0.16779585182666779, -0.4214591979980469, -0.37413132190704346, 0.43514561653137207, 0.16819903254508972, 0.7814382314682007, -0.20421116054058075, -0.0038356047589331865, 0.11027687042951584, 0.6696363091468811, -1.4491320848464966, 0.19423310458660126, -0.05683278664946556, 0.7207713723182678, -0.9496418833732605, -1.0077705383300781, -0.5968700647354126, -0.028399981558322906, 0.7928248047828674, 0.22889883816242218, -1.1188466548919678, -0.6440182328224182, 0.6532660126686096, 1.4086302518844604, 0.07627470046281815, -0.8585742115974426, 0.20224063098430634, -0.5413571000099182, -1.2697359323501587, 1.0982208251953125, -0.14073413610458374, 0.28770068287849426, -1.7136257886886597, -0.40925538539886475, 0.6148024201393127, -0.4533270597457886, -0.23303698003292084, 2.0423386096954346, -1.0297281742095947, -0.8678867220878601, 0.2730708420276642, 0.9837522506713867, -0.4609626829624176, -1.020012378692627, 1.1570974588394165, 0.7619825005531311, 0.773176372051239, 0.6660469770431519, 1.6040023565292358, 0.6121339201927185, 0.66154944896698, -1.384795069694519, -0.1052161455154419, -0.0895981416106224, 0.6411675810813904, 0.7782564759254456, -0.016149653121829033, 0.1072121188044548, -0.5257972478866577, 1.2281590700149536, -0.7139047384262085, 0.15352687239646912, -2.476085662841797, 0.39201322197914124, -2.2697179317474365, -1.2280579805374146, 1.702622652053833, 0.8541083931922913, 1.0679980516433716, -2.8943536281585693, -1.8385509252548218, -0.3569878041744232, -1.1654114723205566, 0.09769392758607864, -0.359950989484787, -0.03257220238447189, 0.6892731785774231, -0.8928927779197693, 0.0515495128929615, -0.12567149102687836, 0.5875986218452454, 1.1954127550125122, 0.8814269304275513, 0.12713512778282166, 0.17103905975818634, 0.45150381326675415, -1.0424758195877075, -0.4762720763683319, -0.5711147785186768, -1.8262708187103271, -0.03632170706987381, 0.13296367228031158], [-0.4153098464012146, 0.08946952223777771, 1.141203761100769, -0.9468303322792053, -0.10679991543292999, -1.9196577072143555, 0.5874086618423462, -1.710819959640503, 0.30946967005729675, -0.8283117413520813, 1.4494190216064453, -0.0915389135479927, -0.6741134524345398, -0.36858972907066345, 0.044339943677186966, 0.3163285255432129, -0.7014951109886169, 0.05609341710805893, 0.5940307378768921, -1.0288612842559814, -0.2465711534023285, -1.3048782348632812, 0.24224890768527985, 0.21252161264419556, -0.9346742630004883, -1.0136164426803589, 0.8561862707138062, 0.3713926374912262, -0.05188295245170593, 0.7375897169113159, 0.09760770201683044, -1.0432809591293335, 0.0348876416683197, 1.3961505889892578, -1.030921459197998, 0.18663841485977173, 2.4954674243927, 1.0533565282821655, -0.6459572315216064, -1.1560205221176147, -0.563275158405304, -1.5739145278930664, 2.362320899963379, 0.6605736613273621, -0.36909812688827515, -1.2708613872528076, -0.024618366733193398, -0.5797237157821655, -0.8028209209442139, 0.3620671331882477, -0.22011451423168182, -1.2760854959487915, 0.9242066144943237, 1.1175975799560547, -0.06269896030426025, 0.9733220934867859, -0.05630692094564438, -1.0721873044967651, 0.7336218357086182, 0.15369287133216858, 0.37396934628486633, -2.254624843597412, 0.5860451459884644, -0.6712955236434937, -0.6114338636398315, 0.6040242314338684, 0.7712996006011963, 0.5036115646362305, -0.13944241404533386, -2.079420804977417, 0.23437125980854034, -0.06468106806278229, 1.3210610151290894, -0.4306166470050812, -1.7403002977371216, -2.3168785572052, -0.5776168704032898, 0.48426127433776855, 0.524503767490387, -2.1694507598876953, 0.6453719735145569, 0.12659089267253876, -0.8453257083892822, 2.3659520149230957, 1.474977970123291, 0.2934530973434448, -1.4519129991531372, 0.41530081629753113, 1.5613945722579956, -2.396268606185913, 0.027414187788963318, -1.3291127681732178, -0.11126727610826492, -1.262008786201477, 1.0990430116653442, 0.4404567778110504, -0.35755112767219543, -0.2030106782913208, -0.24196307361125946, -0.8442224860191345, 0.43780916929244995, -0.5485925078392029, -0.9971822500228882, 1.5210936069488525, -0.05918744578957558, -1.5870033502578735, -0.3714173436164856, -0.34622785449028015, -0.7632506489753723, 0.9256674647331238, -3.676565408706665, -0.371568500995636, -0.6763830780982971, 0.990644633769989, -1.1390656232833862, 0.3932783007621765, -0.7677229046821594, 1.1898335218429565, -0.09427987039089203, -0.905756413936615, -1.16669499874115, 2.554708957672119, -0.5720217227935791, -1.0504499673843384, -1.2630597352981567, -1.7610726356506348, 0.6518604159355164, 0.8618940114974976], [0.4838707149028778, -1.1042898893356323, -0.4392187297344208, -0.22098438441753387, -2.025383472442627, 0.4938288927078247, 0.011347406543791294, 0.9640204906463623, -0.14627625048160553, 0.7303602695465088, 1.6557177305221558, -0.5297216176986694, -0.4239386022090912, -1.5359431505203247, 0.43121036887168884, 0.5015302300453186, 0.03469874709844589, -0.7643928527832031, -0.28828585147857666, -1.5619415044784546, -0.7624968886375427, 0.6991309523582458, -1.9067630767822266, -0.6386339664459229, -1.0279754400253296, 0.5677012205123901, 0.4531717896461487, -0.07167200744152069, 0.12081008404493332, 0.017606012523174286, -0.20649760961532593, 1.1685518026351929, 0.5785908102989197, -0.7157228589057922, 0.002744122641161084, 1.2629621028900146, -1.5024304389953613, 0.47704240679740906, 0.3032848834991455, -0.3830394148826599, -0.11759689450263977, 1.2533122301101685, -0.07685340940952301, 0.6544131636619568, 1.1124285459518433, -0.041473422199487686, -0.05794882774353027, 0.024079829454421997, -0.7157328724861145, -0.06200899928808212, -0.5986896753311157, -0.3302421271800995, -0.6685652136802673, -0.10667908936738968, 1.465835690498352, 0.8556123375892639, -1.3740308284759521, 0.06115978583693504, -1.187727928161621, 0.5546932816505432, -0.011014890857040882, 0.850555956363678, -0.22915621101856232, -0.8363286256790161, 1.4404807090759277, -0.7521394491195679, 1.908991813659668, 0.14965660870075226, -0.16585668921470642, 0.47308555245399475, 0.6392795443534851, -0.8117702007293701, 0.19341260194778442, -0.21191652119159698, -0.5026574730873108, 0.554283618927002, -0.6483007669448853, 0.6196282505989075, -1.2168465852737427, -0.4181704521179199, -0.9160300493240356, 2.7365994453430176, 0.07971161603927612, -0.922635555267334, 1.3997306823730469, -0.07694894075393677, -1.5640065670013428, -0.7586762309074402, 0.09976179897785187, 0.5594790577888489, 0.18757960200309753, 3.102139711380005, 0.2665037214756012, -0.3388833999633789, -1.5094189643859863, 0.14706115424633026, 0.04873969405889511, -0.16919374465942383, -1.0447300672531128, 0.4793207347393036, -1.3967952728271484, -1.4328302145004272, -1.7032057046890259, -1.4804768562316895, 0.962732195854187, -2.466576099395752, 0.20984862744808197, 0.01089952327311039, -1.5782709121704102, -1.0464396476745605, -0.9638025760650635, -1.9422551393508911, 1.1872881650924683, 2.1194872856140137, -0.44680169224739075, 0.44824379682540894, -1.4453132152557373, -0.4147922396659851, 1.9329084157943726, -0.3092098534107208, 0.09129222482442856, 0.1316412091255188, 2.4045302867889404, -0.2486071139574051, -0.803092896938324, 0.24895039200782776, -1.1086516380310059, -0.8532854914665222], [-0.4868137538433075, -0.17013250291347504, -0.16765454411506653, 0.10430783033370972, -0.7082282304763794, 0.3998446464538574, 1.7744498252868652, 0.7540920376777649, 0.19107207655906677, -0.6860674023628235, 1.340855598449707, 0.46605485677719116, 0.03892204910516739, 1.70653235912323, -0.13738302886486053, -1.2604742050170898, -0.739965558052063, 0.3897268772125244, -1.0493651628494263, 0.6590180397033691, -0.897473931312561, 2.0229198932647705, 0.4590993821620941, -1.3089849948883057, 0.16726288199424744, -0.705479085445404, 0.4501813054084778, 1.2375414371490479, -0.9648573994636536, -2.196568250656128, -0.1488284468650818, 0.10088122636079788, 1.4690451622009277, 0.7844692468643188, -0.17955918610095978, -0.0761025920510292, 0.5161503553390503, 1.5630840063095093, 0.41234466433525085, -0.2327861487865448, -0.5152439475059509, 0.9903426170349121, 1.485689640045166, -1.2608331441879272, -0.509309709072113, 0.052519217133522034, 0.39222195744514465, -0.16126728057861328, 2.261458158493042, 0.20625749230384827, -2.4295084476470947, -0.15335366129875183, 1.7660574913024902, -1.2991502285003662, -0.0448555126786232, 1.8191477060317993, 0.4929179847240448, -0.48910537362098694, 1.3937100172042847, 0.2649461328983307, -0.031828779727220535, 0.877677857875824, -0.2163219451904297, -1.103721261024475, 0.6737138628959656, -0.21321609616279602, -1.2183083295822144, -0.5594040155410767, 0.23947857320308685, -2.030251979827881, -0.21245695650577545, -0.9607701301574707, 0.2605515122413635, 1.3265295028686523, -1.2645716667175293, 0.9316990375518799, 0.39613571763038635, -0.13099981844425201, -1.1649959087371826, 0.8529151082038879, 0.8332352042198181, 0.6490631103515625, -1.3677717447280884, -1.945617914199829, 0.7377487421035767, -0.5289825797080994, 0.7656707763671875, -1.247709035873413, -0.05720097944140434, 1.897933006286621, 0.4967271685600281, 0.7972607612609863, 0.2053034007549286, 0.32766208052635193, 0.582629382610321, -0.3715263903141022, 0.6460086107254028, 2.090198040008545, -0.7640935778617859, 1.3283265829086304, 1.6708780527114868, -1.0326430797576904, 0.24000734090805054, 1.8620949983596802, 0.3124385476112366, 1.4095195531845093, -0.6159989237785339, 0.34329041838645935, -0.4279715418815613, 0.15484702587127686, -0.011417508125305176, -0.8798989057540894, 0.5579429864883423, -0.13487015664577484, -1.101911187171936, -0.981955885887146, -0.5994408130645752, 0.9546995759010315, 0.15186648070812225, 0.8290058970451355, 0.09077631682157516, -0.011748891323804855, 0.766736626625061, -0.3660060465335846, -0.09212809056043625, 0.07611603289842606, -1.6886464357376099, -0.5987695455551147], [-1.4896564483642578, -0.24167852103710175, -0.6500977873802185, 0.4570785164833069, 0.9731053113937378, 1.8077785968780518, 0.38475939631462097, -0.6857045292854309, -0.06212081387639046, -1.493507742881775, 0.24934101104736328, 2.1320600509643555, 0.0027522945310920477, 0.3782846927642822, -1.3772387504577637, -0.26058539748191833, -0.24510276317596436, -0.03200672194361687, -0.2146691083908081, 0.18574360013008118, -1.2719769477844238, -1.1865556240081787, 0.590042233467102, 0.8652328848838806, -1.4157521724700928, -0.00258127530105412, 0.018608709797263145, 0.7294019460678101, -0.2625880539417267, -2.3920931816101074, 0.4140913784503937, -0.4961059093475342, -2.138002634048462, 0.9031932950019836, 1.9273381233215332, -0.28376248478889465, -0.10463371872901917, -0.3330594301223755, -0.2552242577075958, 0.035352081060409546, 1.38712477684021, 0.9344351887702942, -0.8070129752159119, -1.8537226915359497, -1.4410687685012817, -1.9848295450210571, -0.15280234813690186, 0.21759356558322906, 1.5990979671478271, -0.012852574698626995, 1.2059794664382935, 0.7192323207855225, 0.31491991877555847, 2.1279730796813965, 0.11082891374826431, -0.07496507465839386, 1.8006656169891357, -0.2650228440761566, 0.6650441884994507, -1.4385051727294922, -0.5525602698326111, -0.6974650025367737, 0.541187047958374, 0.94908207654953, 0.41629815101623535, 0.8716964721679688, 2.5373330116271973, 1.303526520729065, -0.2117832750082016, 0.8771044611930847, -1.3636276721954346, -1.0784761905670166, 0.22671152651309967, 0.33037588000297546, -0.998534083366394, 2.498863697052002, -0.5442246794700623, 0.9657295346260071, 0.20770792663097382, 0.6112193465232849, 1.1922038793563843, 2.406006097793579, -0.556727409362793, 0.4251355528831482, 0.24028946459293365, 0.6851707100868225, -1.6808916330337524, 1.8668458461761475, 0.0798652172088623, 0.5736140608787537, 0.4385422170162201, 1.334212303161621, -0.16239787638187408, 0.4232698678970337, 0.9074589610099792, -0.0644131675362587, 0.7816205620765686, 2.507678270339966, -0.3274681866168976, 0.4250938892364502, 2.135007858276367, 0.5285333395004272, -1.3423597812652588, 0.2979462742805481, -0.05666319653391838, 0.6511585116386414, -0.37503188848495483, -0.3854202628135681, 0.32745835185050964, -1.2219972610473633, -1.5778143405914307, 0.47406280040740967, 1.524743914604187, 0.8358551859855652, -0.4986169934272766, -0.6229773163795471, 1.6575672626495361, -0.17081816494464874, 0.2590298652648926, 0.6769309639930725, 0.69100022315979, -1.4076346158981323, 0.004784812219440937, -0.05469220504164696, -0.14969147741794586, 0.67539381980896, -1.3600668907165527, 0.2112027406692505], [0.16928695142269135, -2.4831767082214355, -1.6839678287506104, 0.6447790861129761, 0.2451929748058319, -2.2207579612731934, 0.005509461276233196, 1.2699530124664307, 1.3693844079971313, 0.16927874088287354, 0.10013128072023392, 0.4028508961200714, 0.4911675453186035, -0.9677478671073914, -1.0621329545974731, 1.1687016487121582, 0.24551288783550262, 0.982064962387085, -2.406430244445801, 0.8129143714904785, -0.4343697428703308, -0.4707954525947571, 1.3669469356536865, -0.3949050009250641, 1.5194299221038818, 2.669506788253784, 0.9461994171142578, 0.8916197419166565, 0.07882486283779144, 0.2658299207687378, -0.11792658269405365, -0.9413959383964539, -0.5667180418968201, 0.33652982115745544, -0.5807943344116211, 0.7982211709022522, 0.5083645582199097, 0.03993714600801468, 0.04203220084309578, -0.38697272539138794, 0.685691237449646, -1.4652599096298218, -0.41046586632728577, -2.109180450439453, -0.029827263206243515, 0.3371126353740692, -0.00024412028142251074, 0.3012763261795044, 0.7403980493545532, 0.8735666275024414, 1.5585013628005981, 0.2758813202381134, -0.03856789693236351, 0.11033208668231964, 0.6020841002464294, 0.2084202617406845, 0.6457250118255615, 2.027616262435913, 0.19956767559051514, 2.390913963317871, -0.5480151176452637, -0.9143637418746948, 0.40658941864967346, -1.345799207687378, -2.428725004196167, -0.5413163304328918, -0.23559893667697906, -0.17913208901882172, 0.2910107970237732, 0.42313772439956665, -1.3763272762298584, -0.5392293334007263, -0.878448486328125, 0.5081519484519958, -0.4053417146205902, 0.5720625519752502, 1.2342579364776611, 1.4523909091949463, -0.06674370169639587, 1.1352014541625977, -2.620083808898926, 0.4405961334705353, -1.1105332374572754, -0.3928743600845337, 0.953525722026825, 1.1448556184768677, -1.6744648218154907, -0.43665096163749695, -0.1976323276758194, 1.3667724132537842, 0.7833513617515564, 1.6240627765655518, 0.5189031362533569, -2.4923605918884277, 0.06964050978422165, 0.7808114290237427, -0.9832247495651245, -0.6710816025733948, -0.5811695456504822, 1.422752857208252, -0.5804827809333801, -1.3281776905059814, 0.29098162055015564, -0.0035569295287132263, -0.9941650032997131, -0.052949510514736176, -0.5479521751403809, 1.1624804735183716, -0.6791669130325317, 2.2772204875946045, 1.8770852088928223, 1.8116568326950073, -0.9383693337440491, -0.9974289536476135, -0.5912306904792786, 0.5776275992393494, -1.197838544845581, 0.9535843133926392, -0.7569620609283447, 1.1969023942947388, -2.270413398742676, 2.3433353900909424, -0.4441203773021698, 0.3621537387371063, -0.43078917264938354, -0.10332565754652023, 1.1317322254180908, 0.15963031351566315], [0.5453232526779175, -1.815483570098877, -0.6960895657539368, 1.567298412322998, -0.07243777066469193, -0.54636150598526, -0.18639425933361053, -0.11913835257291794, -1.3939199447631836, -0.6926465034484863, 0.33607178926467896, 0.16425803303718567, 0.32249531149864197, -0.35623180866241455, 0.15177206695079803, 0.7042546272277832, -0.7611901760101318, -1.2432279586791992, 0.20293311774730682, -1.4418182373046875, 1.3491699695587158, 0.6007348299026489, 0.9364913105964661, -0.6906372308731079, -0.5753207206726074, 0.13724252581596375, 1.1918203830718994, 0.12751628458499908, 0.17650392651557922, 0.25958123803138733, 2.9758493900299072, -0.16145870089530945, -0.32503753900527954, -1.641356348991394, 0.5941392183303833, 0.24220681190490723, -2.7585630416870117, 0.0993446335196495, -1.812612533569336, 0.04319421201944351, -1.1247782707214355, -0.9039769768714905, 0.149897500872612, -0.3972640037536621, 0.3442118465900421, 1.3162809610366821, 0.8400353193283081, 0.05712065100669861, 1.1686955690383911, 0.3792717158794403, 0.10784734785556793, 0.10444952547550201, 0.7321494817733765, 0.19893993437290192, -0.2264672815799713, -0.09757999330759048, 0.8350509405136108, 2.368689775466919, -0.10246913880109787, 0.6360069513320923, 1.4295597076416016, -0.245492622256279, 0.3334025740623474, 0.4362340569496155, 0.5792438983917236, -0.5269665122032166, 1.5381758213043213, -1.0165610313415527, -1.1755225658416748, -1.1987779140472412, -0.12972503900527954, 0.5297259092330933, -0.9463954567909241, 1.6235733032226562, -0.04428669810295105, -0.008892270736396313, -0.5788196325302124, -1.6116135120391846, 1.0283983945846558, 1.6812182664871216, -1.0055909156799316, -1.1661567687988281, 2.53592848777771, -2.0929548740386963, 1.4603079557418823, 1.114245057106018, 0.0011718407040461898, 1.7372353076934814, -1.0050584077835083, -0.19859609007835388, 1.1397693157196045, -0.7205429077148438, -0.8166181445121765, 0.14181700348854065, -1.6643991470336914, -0.7282513976097107, 0.27073460817337036, -0.8856858015060425, 1.7453043460845947, -0.8163242340087891, -0.6662157773971558, 0.29140201210975647, 1.329397201538086, -1.3280212879180908, 0.07096979767084122, -1.5046762228012085, -1.212929606437683, 0.08298035711050034, 1.1710247993469238, -1.1673723459243774, -0.21866612136363983, 0.8597528338432312, -1.6009416580200195, 0.2416907250881195, 0.7299624085426331, -0.9711588621139526, -1.4241255521774292, 1.2362430095672607, -0.047507479786872864, -0.14778463542461395, 1.8282524347305298, 0.298576682806015, 0.5100271701812744, 1.3191289901733398, -0.9994471669197083, 1.0613797903060913, 2.1452457904815674, -1.1721776723861694], [-0.8261048793792725, 0.3081543743610382, 1.0644158124923706, 0.2607176601886749, -0.6272707581520081, 1.533403754234314, 1.0432568788528442, 2.0701754093170166, 0.3525901436805725, 0.4317319095134735, 0.004361970815807581, -1.8550113439559937, 0.7659482359886169, 0.5868254899978638, -1.4444533586502075, -1.497053623199463, -0.6451149582862854, 0.6993755102157593, -0.10982612520456314, -1.1695767641067505, 0.5788390636444092, 1.0954248905181885, 0.28992345929145813, -0.8094128966331482, 1.0903061628341675, 0.8441647291183472, -0.5795580148696899, 0.773917019367218, -1.3796919584274292, -1.4971600770950317, -1.391982913017273, 0.5055978298187256, 1.3549315929412842, -0.815515398979187, 0.21140718460083008, 1.0615730285644531, 0.23757094144821167, 1.3486305475234985, 1.3807905912399292, 0.14996084570884705, 0.08830683678388596, 1.5151439905166626, -0.5195848941802979, -0.5589275360107422, -0.2890275716781616, 2.7855846881866455, -1.2815998792648315, -0.8119180798530579, -0.9974632859230042, 0.9158757925033569, -0.4344393014907837, 1.0461808443069458, 0.37641751766204834, -2.132146120071411, -1.5105066299438477, 0.12659160792827606, -0.31700342893600464, 0.1596105843782425, -1.657344937324524, 0.44999659061431885, 0.5451391935348511, -0.4847927391529083, 0.6259570717811584, 0.32842063903808594, -0.2898007333278656, -0.39248478412628174, 1.55245840549469, -0.20193926990032196, 1.3633298873901367, -0.4440693259239197, -0.5353456735610962, -0.4265732765197754, -1.327116847038269, 1.289608359336853, -0.5769439935684204, -0.41692233085632324, 1.6368635892868042, -0.018631137907505035, 0.29426926374435425, 0.7114923000335693, 1.2222096920013428, -0.7865070104598999, -0.7094928622245789, 0.7789052724838257, 0.5509126782417297, 1.0937812328338623, 0.7482947707176208, 0.8945977687835693, -0.2500017583370209, 1.6919678449630737, 0.4630855917930603, 0.7167940139770508, 2.4200587272644043, -0.7108697295188904, -0.4207831025123596, 1.0076297521591187, 1.4909884929656982, -0.8877121210098267, 0.06671813130378723, -0.2237740159034729, -0.3716481029987335, -1.6688828468322754, 0.11327744275331497, 0.9903875589370728, -0.25016042590141296, -0.07268427312374115, 2.006704807281494, -0.15134580433368683, -0.14268681406974792, -0.8608483076095581, -0.24162274599075317, 0.8243441581726074, 0.21259629726409912, -0.3302500247955322, 2.5378549098968506, 0.4423374533653259, 0.9888971447944641, -0.09573046863079071, -0.17425674200057983, 1.2201876640319824, -0.33915090560913086, -2.2083733081817627, -0.08592752367258072, 0.35037267208099365, 0.11935645341873169, -0.039544958621263504, 0.2815566658973694, 0.8254660964012146], [-0.833075761795044, -0.9597572684288025, -0.6071398258209229, 0.3166397511959076, 0.9266406893730164, 0.4372674226760864, 0.34028133749961853, -1.2674320936203003, -0.6649677157402039, 0.2252984195947647, 0.27135390043258667, 1.8986873626708984, -0.7194719314575195, 1.0821505784988403, 0.6276437640190125, 1.2223843336105347, -1.4351216554641724, -0.3201742172241211, 0.5398643612861633, 0.1989317536354065, 0.6319535970687866, -0.36370840668678284, 0.048596516251564026, 0.22865243256092072, -1.0708004236221313, -0.9960131049156189, 0.8498736023902893, 0.036208923906087875, -1.4834591150283813, 1.5795480012893677, -0.6627835035324097, -0.6885837912559509, 1.9557040929794312, -0.28328776359558105, -1.3452776670455933, 1.19999098777771, -0.10600363463163376, -1.403353214263916, 0.24476245045661926, -0.15829747915267944, 2.9617698192596436, 0.6878650188446045, 0.7113451361656189, -0.5399623513221741, 1.066968321800232, 0.8130788207054138, 2.3521947860717773, -0.6062409281730652, -0.9251431226730347, -0.9783487915992737, -0.7190587520599365, 1.8858263492584229, 0.7962130904197693, -1.1013885736465454, 0.7048488259315491, -0.18949730694293976, 1.324371337890625, -0.23426453769207, -0.27376219630241394, 0.2968461215496063, 0.15578752756118774, 0.39790189266204834, 0.8814275860786438, 1.4708592891693115, -0.831453263759613, -0.49356135725975037, 0.7149316668510437, 0.5634313225746155, -0.7268074750900269, -2.0986828804016113, 1.4396227598190308, 0.32466447353363037, 0.7721675634384155, -1.5665642023086548, 3.1490259170532227, -0.6861559152603149, -0.7259999513626099, -0.10127155482769012, 1.8967682123184204, -0.5460783243179321, -1.4322417974472046, 0.294635146856308, 1.584219217300415, 1.4398916959762573, 0.43710657954216003, 2.1508452892303467, -0.7547243237495422, 0.1716158092021942, -1.09048593044281, -1.7022678852081299, 3.181063652038574, -0.1116708517074585, 0.7775635719299316, 0.18382200598716736, 0.8986465930938721, 0.9520323872566223, 0.5932614207267761, -1.0248522758483887, -0.9995847940444946, -0.10702010989189148, 1.64812433719635, 0.18365907669067383, -0.3850044012069702, 1.1292723417282104, 0.21093899011611938, 0.6978432536125183, -0.19539690017700195, 0.4576154947280884, 0.17364948987960815, -0.550942599773407, 1.1139742136001587, -0.09507998079061508, 0.2682957351207733, -1.9024053812026978, 0.05426062270998955, -0.514556348323822, -0.18052929639816284, -2.470477342605591, 1.073699712753296, -0.741704523563385, 1.2275738716125488, 1.2286934852600098, 0.8425689935684204, -0.17337748408317566, -0.10788343101739883, 0.8381758332252502, 0.9274640679359436, -0.555582582950592], [-0.42621710896492004, -0.11989984661340714, -0.8845435380935669, 1.0333677530288696, 1.0884898900985718, 0.14426235854625702, 0.1617957204580307, -0.23678922653198242, 1.4771525859832764, -1.5944879055023193, -1.3509780168533325, -0.7915992736816406, 0.5105772614479065, -0.10874484479427338, -0.4361982047557831, -2.0273404121398926, -0.9837099313735962, 1.0030839443206787, -1.4077235460281372, -0.09682139754295349, 0.4984305799007416, 0.5187281370162964, 2.1711349487304688, 1.3529915809631348, -1.6721537113189697, 0.1700575053691864, -0.5163642764091492, -0.736849308013916, -0.1851383000612259, -1.7586801052093506, 0.0566815584897995, 2.1953985691070557, -0.2557881474494934, -0.929825484752655, -1.198227047920227, 3.089674711227417, -2.401995897293091, -0.48952749371528625, -0.8626078367233276, -1.1646100282669067, 0.6589593291282654, 0.2257784754037857, 0.8497634530067444, -0.5703262090682983, -0.29289060831069946, 0.29976916313171387, 1.155085802078247, -0.874323844909668, 0.06729188561439514, 0.13619090616703033, -0.016257086768746376, 0.06605371832847595, -0.5537993907928467, 1.1183531284332275, -0.76798015832901, -0.6948880553245544, 0.3115396797657013, 0.26350539922714233, 0.2407289296388626, -0.7263760566711426, -0.5696288347244263, -0.09935420751571655, -0.8323805332183838, 0.014258656650781631, 0.2759643793106079, -1.5694019794464111, 1.6102118492126465, 0.7686373591423035, -1.807058334350586, 0.4230731129646301, -2.300708532333374, 2.036957025527954, 0.9077573418617249, -1.8425571918487549, -0.11934466660022736, 0.7637050747871399, -0.29812613129615784, 1.2514394521713257, -1.0705866813659668, 0.682604193687439, -0.5443567633628845, -1.6049563884735107, -1.1403253078460693, -0.4876902401447296, 0.5217891931533813, -1.3257302045822144, 1.347032904624939, 2.0464420318603516, 0.28241533041000366, 1.638061761856079, -0.2333202064037323, -0.030522046610713005, 1.7460335493087769, 0.12004824727773666, 0.9975029230117798, -0.9306139945983887, 0.3477135896682739, 0.49862807989120483, -0.5596521496772766, 0.1453174650669098, 0.11446237564086914, 0.232639878988266, 0.12392479181289673, 0.4259573221206665, -1.4092624187469482, 0.09359102696180344, -0.39138439297676086, -0.36670684814453125, 0.42758315801620483, 2.1027488708496094, 0.10142973810434341, -0.43207263946533203, 0.1016250029206276, -0.005574893206357956, -0.2108258605003357, -1.3447341918945312, -0.267399400472641, -0.20713143050670624, 0.044452980160713196, 0.126173198223114, 1.2327640056610107, -0.9121478796005249, -0.60736083984375, -0.45141714811325073, 0.7279302477836609, 0.2815588116645813, -1.5312503576278687, -0.35409238934516907], [-0.06614743918180466, 0.2281375378370285, -0.7671670317649841, -0.44766291975975037, 0.6537735462188721, -0.044102832674980164, 1.8848427534103394, 1.821044921875, -1.0467602014541626, 0.0291171632707119, -0.11445143818855286, -0.4519534111022949, 0.3122207820415497, -1.1825971603393555, -0.631406843662262, -0.21582333743572235, 0.2723294496536255, -0.04153002053499222, -0.1451873630285263, 0.9113369584083557, -1.1596966981887817, 0.556545078754425, -0.9698084592819214, 0.3203960657119751, -0.5903647541999817, 1.0489193201065063, 0.010055556893348694, 0.5790379643440247, 0.6907095313072205, -0.4348679184913635, 0.6517065167427063, 0.9940676689147949, 0.6374779343605042, 1.0575438737869263, -0.12227597832679749, 1.116502285003662, -0.5607845187187195, -0.07911407947540283, 0.7104536294937134, 0.3985942006111145, -0.7508053183555603, -0.18726274371147156, -0.8924145698547363, 0.0039453632198274136, -0.22464993596076965, -0.6330054402351379, -0.19645139575004578, -0.20820602774620056, -0.8457380533218384, 0.43213626742362976, 0.04215698316693306, -0.36532023549079895, -0.19174106419086456, 0.3535161018371582, -0.2566734254360199, -0.3980283737182617, 0.5635385513305664, -0.24762384593486786, -0.39011380076408386, 1.7348822355270386, 1.6318753957748413, 1.0561044216156006, -0.00795541238039732, -0.9065543413162231, 0.4491264224052429, -0.41414329409599304, 0.5193573236465454, 2.010279417037964, 0.7664074897766113, -1.6534538269042969, 1.1617050170898438, -1.2978445291519165, -0.28261345624923706, -0.9818887710571289, -1.1802723407745361, -0.02493569813668728, -0.9641836881637573, -0.794969916343689, 0.2505498230457306, 0.03306414186954498, -0.6222915649414062, -1.4688690900802612, 1.491924524307251, 1.4303371906280518, -0.49658140540122986, 0.027450909838080406, 0.41538307070732117, 1.2147612571716309, 0.6603150367736816, -1.0308654308319092, 0.034155189990997314, -1.5016132593154907, -0.009636186994612217, 0.717025637626648, 0.7611415386199951, 1.0836951732635498, 1.2002043724060059, -0.45549431443214417, -0.8245460391044617, 1.4451733827590942, -1.5849483013153076, 1.2178237438201904, 0.08525747805833817, 0.7993519306182861, -0.48035871982574463, -1.232784390449524, -0.547059953212738, -1.3745777606964111, -0.010491381399333477, 0.13615548610687256, 0.5983606576919556, 0.26674798130989075, 0.08860129863023758, -0.5301012992858887, -0.5237334370613098, -1.1644076108932495, 0.42021071910858154, -1.0675837993621826, -0.26852378249168396, -0.5358497500419617, -0.004489891696721315, -0.13028690218925476, 0.5642378926277161, -0.4879302382469177, 1.1239657402038574, 0.6740725040435791, 0.31867724657058716, -0.17648105323314667], [1.7309659719467163, 0.5318702459335327, -1.1432127952575684, -1.0955684185028076, -0.6170070171356201, 0.2890474200248718, 1.2004177570343018, 0.39202460646629333, 0.8268871903419495, -1.0319916009902954, -1.0185109376907349, 0.37255266308784485, 2.2450315952301025, -1.435699701309204, -0.3082905113697052, -0.9825969338417053, 0.8656454086303711, -1.5120456218719482, -0.0009679130744189024, 0.5095992088317871, 0.326600581407547, 0.512220025062561, -0.1313977986574173, 1.4525034427642822, -0.1799912303686142, 0.6723796129226685, -0.542682409286499, 0.9615965485572815, 0.037754226475954056, -0.45535069704055786, -1.7431412935256958, -0.6973673105239868, -0.920278787612915, 1.238200068473816, 1.467488169670105, -0.26742076873779297, 0.3795206546783447, 0.25479722023010254, 0.23853546380996704, -0.0641709715127945, 0.4327099919319153, -0.42660412192344666, 0.3184133768081665, 0.027745593339204788, 2.089818000793457, 0.27396729588508606, 0.2581576704978943, -0.846011221408844, -0.3959866464138031, 0.222430020570755, -0.7173979878425598, -0.3266722559928894, -1.2112023830413818, 1.2452176809310913, -1.8328436613082886, -2.1466102600097656, 1.0530695915222168, 0.6070660352706909, -1.6619714498519897, -2.1204726696014404, 0.5775477290153503, 0.8521772623062134, 1.3787962198257446, -0.8534472584724426, -0.10190923511981964, 0.8707864880561829, 2.2515978813171387, -2.008175849914551, 0.8921627998352051, 2.3361167907714844, 0.26871371269226074, -0.28064385056495667, 0.7936705946922302, -0.8851034045219421, 1.7925512790679932, 0.26717010140419006, 2.37420916557312, 0.2236345112323761, -1.6437768936157227, -0.682161808013916, -0.6972464323043823, -0.41019555926322937, -1.3549851179122925, -0.5866924524307251, -0.42692112922668457, 0.18257929384708405, -0.598311185836792, -0.22386519610881805, 0.8333139419555664, 0.7936558723449707, 0.2772519290447235, -1.5888813734054565, -0.11519303172826767, -0.2724185883998871, 0.44877657294273376, 1.4521416425704956, -0.891485869884491, -0.09494350105524063, -0.4768766164779663, 1.566066861152649, 0.17668145895004272, 0.8788750171661377, 0.5684989094734192, -1.0559333562850952, 0.2622734308242798, -0.9700539708137512, 1.8055446147918701, -1.0151879787445068, -0.2046935111284256, 0.407807320356369, -0.7658882737159729, 0.7434327602386475, 0.9520272612571716, -0.2917430102825165, -0.6995131373405457, 0.5645948648452759, 0.05235765501856804, -0.43385663628578186, -0.10062412917613983, -0.883303165435791, 2.136730194091797, 0.6968082189559937, -1.3611918687820435, 0.19002243876457214, -1.505765676498413, 1.0381237268447876, 0.4509339928627014, -0.8468934893608093], [-1.5356632471084595, 0.419216126203537, -1.1765097379684448, -0.13947342336177826, -1.4688876867294312, 0.6908449530601501, 0.4646095037460327, 0.23065288364887238, -0.4586654007434845, 0.35564082860946655, -1.5259394645690918, 0.4965287148952484, -0.04417532682418823, 0.3565312623977661, -1.3331154584884644, -1.0437906980514526, 1.489292860031128, 0.19427570700645447, 0.8571264743804932, -1.215408444404602, 0.4533463418483734, -0.6142604947090149, 0.2645748555660248, 0.4125073552131653, -1.5472387075424194, -1.383949875831604, 1.929018259048462, -0.1430703103542328, 2.064506769180298, 0.680357038974762, -0.3777473270893097, 0.03690283000469208, -0.759106457233429, -0.5473169684410095, -1.375909686088562, -0.7097340226173401, 0.8386027812957764, 0.724811315536499, -1.4098904132843018, -0.4631374776363373, -1.9835959672927856, 1.005367398262024, -0.902906596660614, 1.568917155265808, -0.43724608421325684, 0.12842677533626556, -0.6100980639457703, 0.10908353328704834, 0.9157769083976746, 0.7142431735992432, 0.19356007874011993, -0.5234003663063049, 0.6486959457397461, 0.49272847175598145, -0.7714799046516418, -0.41623052954673767, -0.146977499127388, 0.7308750748634338, 1.4155340194702148, -0.27097514271736145, 0.6126580834388733, 0.7449122667312622, 0.3293135166168213, -1.6645677089691162, -1.954380750656128, 0.551914632320404, 0.2833154797554016, 1.4997793436050415, 0.2724020779132843, -0.4885977506637573, 0.028025593608617783, -0.36229926347732544, 0.4818086326122284, 0.5138782858848572, -0.6068933010101318, -1.5223244428634644, 1.1107827425003052, 0.52436763048172, 1.2899630069732666, 0.6631059050559998, 1.9798121452331543, 0.9320042729377747, -0.37033119797706604, -1.2892038822174072, -0.16523678600788116, -0.40595850348472595, 1.2043789625167847, -0.5541825890541077, 0.07413787394762039, 0.39632079005241394, 0.2427094429731369, -0.09236208349466324, -0.13263678550720215, -0.8205909132957458, 0.7148134112358093, 2.067361831665039, 0.3750414550304413, 0.3019297122955322, 0.6139846444129944, -0.5959440469741821, -0.15718480944633484, -1.3685288429260254, 1.0467417240142822, -2.4041216373443604, 0.5302258133888245, -1.429772973060608, 0.8650093078613281, -0.7536674737930298, 0.8247542977333069, -0.3661664128303528, 0.973118245601654, 0.06116991862654686, 1.2620820999145508, -1.3064767122268677, -0.48888081312179565, 0.8402102589607239, -0.8127062320709229, -0.10808710008859634, -2.337977170944214, 0.9923623204231262, 0.6723804473876953, 1.0862853527069092, -0.1901673674583435, -1.2521976232528687, -1.1422398090362549, 1.0231809616088867, -0.31761837005615234, 0.9333420991897583], [-0.21242979168891907, 1.2532018423080444, -0.5971648693084717, -0.3880416750907898, 1.0478448867797852, -2.0833182334899902, 1.243178367614746, 1.1655492782592773, -0.39083144068717957, -1.3485249280929565, -0.3028686046600342, -1.0776139497756958, -0.34869998693466187, -0.9497783780097961, -0.8484141230583191, -1.0561860799789429, 0.30135276913642883, 0.3281545639038086, 1.452052354812622, 1.7788904905319214, 1.6495929956436157, -0.05594995245337486, -0.2661466598510742, -0.5179564356803894, 0.5591411590576172, 0.3842236399650574, -0.4979347288608551, -0.6695457696914673, 0.2703632414340973, 1.5367933511734009, -0.561821460723877, 0.8828921318054199, -1.148975133895874, 1.8416658639907837, 0.1901392638683319, 0.7586703896522522, 0.48638293147087097, 0.8936395645141602, 1.6040318012237549, 0.1937030702829361, 0.45577472448349, 1.0340758562088013, -0.19405411183834076, -1.2285919189453125, 0.3309481143951416, 1.047808289527893, 1.4275410175323486, 0.6275375485420227, -0.2548346519470215, -0.6628673076629639, 1.2403336763381958, -0.9686476588249207, 1.5030186176300049, 0.32703009247779846, -0.3028635084629059, -0.15235306322574615, -0.14010421931743622, 2.0397019386291504, -0.7421004772186279, -1.1082053184509277, -0.8219722509384155, 1.9305514097213745, -0.6128718852996826, -0.19888737797737122, 1.4018242359161377, 0.2759280800819397, -0.26312699913978577, -0.7396337389945984, -1.4245893955230713, -0.449148029088974, -1.0079610347747803, 2.315718173980713, 1.0415711402893066, 0.47189244627952576, 0.5941804051399231, -1.1044553518295288, -1.4116780757904053, -1.4524211883544922, -0.29896315932273865, -0.06997690349817276, 0.6379434466362, -2.0199925899505615, -0.9815109968185425, -0.013026205822825432, 0.4352165460586548, 0.3819110691547394, 0.9687578678131104, 0.10712286829948425, 0.5378357172012329, -0.023938000202178955, 1.2922393083572388, -0.0029188685584813356, 0.6352813243865967, -1.4137450456619263, -1.3000956773757935, -1.572144627571106, -0.34320691227912903, 1.20648193359375, 0.7991876602172852, -0.06398502737283707, 1.2208586931228638, 1.325313925743103, -0.728289783000946, -1.6099555492401123, 0.4503582715988159, 0.8513038158416748, 0.33354541659355164, -1.2367455959320068, 0.34841060638427734, -1.2596007585525513, 0.3968057334423065, -0.14919330179691315, -0.7880975604057312, 0.7944005727767944, -0.4379201829433441, 0.5573747158050537, 0.6074897050857544, 0.821292519569397, -1.6551445722579956, -0.052416834980249405, 1.2846755981445312, 0.8126636147499084, -1.2922953367233276, 0.9055734276771545, 1.1043936014175415, 0.042359016835689545, -0.13664887845516205, -0.5468165874481201]]], [[[-0.19086334109306335, 0.14913123846054077, 0.6242122650146484, 2.3475124835968018, 1.1800497770309448, -1.2570894956588745, 0.11658287793397903, -0.6876460909843445, 0.40332573652267456, 1.2370446920394897, -0.20076331496238708, 1.6675658226013184, -1.3495006561279297, 0.527716338634491, 1.7219051122665405, -0.9469210505485535, 0.7106935977935791, -1.0250824689865112, 1.5344412326812744, 0.5147715210914612, 0.08088373392820358, -0.5034893751144409, -1.2097902297973633, -0.3715972304344177, 0.8614429235458374, 0.9081805944442749, 0.2949310541152954, -0.7581104040145874, 0.11520784348249435, -1.1665287017822266, -0.37838274240493774, -0.46816176176071167, -0.06045997515320778, -0.41558998823165894, 0.4005073308944702, 0.5365528464317322, -0.19886799156665802, -0.9305689930915833, -0.7299360632896423, 0.15069952607154846, 0.9442812204360962, 1.7819944620132446, -0.0830320417881012, 0.5878276824951172, 1.6120374202728271, 2.3414909839630127, 0.5354467034339905, 0.3085944652557373, -0.33394116163253784, 0.39151668548583984, -0.36973315477371216, 1.7930212020874023, 0.7389030456542969, -0.10512411594390869, 0.04332412779331207, -0.40234941244125366, -0.2977067530155182, -0.37812942266464233, -0.34495022892951965, -0.2972375452518463, -0.5267840027809143, 1.286641240119934, -0.2546997368335724, -1.4296609163284302, 1.1384825706481934, 0.6355825662612915, 1.4510740041732788, -0.7498584985733032, -1.1153407096862793, 0.8475530743598938, -0.1955736130475998, -1.101728081703186, -0.09455839544534683, -0.46827587485313416, 0.7113152146339417, -1.0257530212402344, -1.040132761001587, 2.4636948108673096, 1.1143572330474854, -1.1345781087875366, 0.5093502402305603, -0.3127146065235138, 0.6730712056159973, 0.4978918731212616, 0.4153318703174591, -0.2181040644645691, 0.6437243223190308, 0.8085343837738037, 0.004674674477428198, -0.15980017185211182, -1.197900414466858, 0.7001563906669617, -1.2293490171432495, 1.0270639657974243, -0.7627209424972534, -1.085262656211853, -0.02671867236495018, -0.59709233045578, 0.8170574307441711, 0.4550810754299164, -0.46862661838531494, 0.49858102202415466, 0.7913408279418945, 0.7682279944419861, -0.6752122044563293, 0.34865304827690125, 0.48673832416534424, 0.47984108328819275, 0.4022310972213745, -0.6370711922645569, -0.007997732609510422, 0.4030044674873352, -0.760200023651123, 1.835989236831665, 1.1307705640792847, -0.6509823203086853, -0.9968770146369934, 0.20116518437862396, 1.1815600395202637, -0.734521210193634, 1.1800471544265747, -0.6358168125152588, -0.300282746553421, 0.9546106457710266, 0.2617945075035095, 1.0510518550872803, 0.09177149832248688, -0.0991889014840126], [-0.4899519383907318, 0.9942073225975037, 2.262718439102173, 1.3307901620864868, -1.2917941808700562, -0.4112683832645416, -1.0320827960968018, 0.06194232776761055, 0.33197543025016785, -1.0336804389953613, -0.6238048076629639, 2.411860466003418, -0.9418412446975708, 0.16629958152770996, 0.9946188926696777, -0.6122615933418274, 0.3778676986694336, 0.18025770783424377, -1.1193076372146606, -0.03898639604449272, 1.377290964126587, -0.28914856910705566, -0.1467890590429306, -1.9017553329467773, -0.28416764736175537, 0.6085625886917114, -0.04092712327837944, -0.3028130829334259, -1.4763652086257935, 0.874163806438446, -0.018581876531243324, -2.086467981338501, -0.03857038542628288, 0.5366869568824768, 0.8138759732246399, -0.15379175543785095, 0.5766763091087341, -0.6812079548835754, -1.4169374704360962, 0.2678612768650055, -1.7600024938583374, -0.9422336220741272, -1.4653233289718628, 0.21851611137390137, -1.3118101358413696, 0.6983613967895508, -1.0376611948013306, 0.15280219912528992, -1.0424113273620605, -2.6152286529541016, -0.2945709228515625, 0.09973838180303574, 0.3364253342151642, -0.5532790422439575, 0.4381820857524872, -0.7636532187461853, -0.35794302821159363, 0.108711838722229, 0.19442704319953918, -0.5405486226081848, -0.17517927289009094, -1.3557817935943604, 0.6427513957023621, -0.5926788449287415, 0.11712028086185455, 0.6923994421958923, 1.2776668071746826, 1.0286924839019775, -0.2747619152069092, 0.0641971156001091, -0.9582346677780151, -0.07758095860481262, -1.231569766998291, -0.7148722410202026, -0.4800483286380768, 0.49679404497146606, 0.2982766926288605, -0.3667207956314087, -1.6328957080841064, -1.9723265171051025, 1.5193589925765991, 1.3403376340866089, -1.565637230873108, -1.4941887855529785, 1.7681021690368652, -0.4766507148742676, 1.5986980199813843, 1.6955573558807373, 0.38401350378990173, -0.7004596590995789, 0.724278450012207, -1.336719036102295, -0.7088220119476318, -0.8809708952903748, 0.7514265179634094, -0.6611387729644775, -1.3568555116653442, 0.5434130430221558, -0.1096433699131012, 0.9234582185745239, 0.043079640716314316, -0.17008638381958008, -1.3367283344268799, 0.06613466143608093, 0.6983998417854309, -0.013368170708417892, 1.3548173904418945, -2.417480945587158, 0.3201478123664856, 1.1095969676971436, -0.42066019773483276, 0.8165793418884277, -1.018484115600586, -0.025684088468551636, 0.032353658229112625, -0.2951335608959198, -0.09842951595783234, -0.6064862012863159, -0.44732484221458435, -1.2465007305145264, 0.8815537095069885, -0.22913941740989685, -0.11967974156141281, 0.8471357822418213, -0.4881228506565094, 0.882350742816925, 0.45839083194732666, -0.057841889560222626], [1.994038462638855, 1.0580973625183105, 0.5112463235855103, -0.59422767162323, -1.0003398656845093, 0.9864643216133118, 1.7319320440292358, 1.6113946437835693, -0.3950342833995819, 1.4121865034103394, -1.294751524925232, 0.5820615291595459, 0.30693739652633667, -0.10662011802196503, -1.6137487888336182, 0.35057532787323, 0.5646810531616211, 0.013309079222381115, 1.3581874370574951, -1.344804286956787, -0.782021701335907, 0.07048443704843521, -0.1901598572731018, -2.274876117706299, 1.3032060861587524, -2.040351152420044, 0.39909616112709045, 1.1596319675445557, -1.348398208618164, 2.5909318923950195, -1.3939862251281738, -0.2457883208990097, -1.347387671470642, -1.537840723991394, -0.18334493041038513, -1.8898719549179077, -0.2448119819164276, -1.468936800956726, -1.0028059482574463, 0.13349372148513794, 0.4534277021884918, -0.6609506011009216, -0.5718913674354553, 1.3272933959960938, -0.13151375949382782, 0.9122148156166077, -1.3144410848617554, -0.9557260870933533, 0.31010425090789795, -0.013410785235464573, -1.0121197700500488, -0.44938528537750244, 1.3039143085479736, 1.6843595504760742, -1.9578843116760254, 0.7694053649902344, 1.0165849924087524, 0.0631445124745369, -0.18686138093471527, 0.09303651005029678, 2.153801679611206, 0.871017575263977, -1.724980354309082, -0.9480916857719421, 0.598986029624939, 0.05889038369059563, 0.3550831079483032, 0.38704249262809753, 0.6808006763458252, -0.2472178339958191, 0.7277960181236267, -0.401784747838974, -0.9202836751937866, -0.8165735006332397, -0.36635100841522217, 1.9362449645996094, 1.7967621088027954, 0.6925775408744812, -0.3924647867679596, 0.38442716002464294, 0.28093448281288147, 1.3369011878967285, 0.06567957252264023, 0.47389259934425354, 0.5250674486160278, 0.7041270136833191, 0.07574538141489029, -0.9699859023094177, -0.7824382781982422, -0.0786648690700531, -0.8376085758209229, -0.5235328078269958, 0.4816431403160095, -0.4224070906639099, -0.32047826051712036, 0.788706362247467, 0.5317168235778809, 0.4066382348537445, -0.5050912499427795, -0.9327787160873413, 1.9062817096710205, 0.9091641902923584, 0.9818289875984192, -1.379437804222107, -0.22893667221069336, -1.2232264280319214, 0.5032505989074707, -0.13032953441143036, -0.18547163903713226, -1.2848488092422485, 0.7513326406478882, -0.5649042129516602, -0.11975103616714478, 0.44819214940071106, 0.7884398102760315, 0.3771046996116638, 1.6409187316894531, 0.3889763355255127, -0.539718747138977, 0.5288879871368408, -0.9478570222854614, -0.18570837378501892, 0.3652558922767639, 0.2102900892496109, 0.177265465259552, 1.079616665840149, -0.7535465359687805, -1.126369595527649], [1.0861529111862183, 0.7166612148284912, -0.9523638486862183, -0.16425444185733795, -0.48940369486808777, -0.1966240257024765, 1.9381248950958252, -0.5642396211624146, 0.726902425289154, 0.7252844572067261, 1.941534161567688, -2.2086164951324463, -0.8034811019897461, -0.20869159698486328, 0.8757798075675964, -1.1243952512741089, -0.3636922240257263, 0.1678164005279541, 0.5140677690505981, -0.6339790225028992, 1.188111424446106, 0.38990992307662964, -0.04400819167494774, 1.3377162218093872, 0.5201404094696045, 0.2532961964607239, -0.14717510342597961, -1.146382212638855, 0.21755582094192505, 0.13843418657779694, -0.3416047990322113, -1.8744869232177734, 1.9559564590454102, -0.3807724416255951, 2.2191739082336426, 0.5786558389663696, -0.5221493244171143, -0.0063885352574288845, 1.7504053115844727, 1.242043137550354, 1.6639595031738281, -1.4167438745498657, -1.269392967224121, -0.3672592043876648, 0.021182117983698845, 0.18204286694526672, -0.513369083404541, -0.4649600386619568, -1.0290042161941528, 0.04700790345668793, 0.3609332740306854, -0.6136611700057983, 0.4448796212673187, -0.3180772066116333, -2.221118688583374, -0.2850835621356964, -1.864131212234497, -1.402832269668579, 0.9020370244979858, 0.5730730295181274, 0.26448118686676025, 0.0934012159705162, 1.2630170583724976, 0.08633638173341751, 1.723157286643982, 0.8237089514732361, -0.07830218225717545, -3.0207672119140625, 0.4570853114128113, -0.4142482876777649, 0.14918352663516998, -0.8068013191223145, -0.8288829326629639, -0.1991664171218872, -1.916425108909607, -1.1858770847320557, 1.5346243381500244, -0.29094430804252625, -0.857757031917572, -0.9464190006256104, 0.11906266212463379, 0.7393965125083923, 0.09973663836717606, -0.8928989768028259, 0.7437447309494019, 1.0907331705093384, 0.3091403841972351, -0.6083170771598816, -0.4829062521457672, 0.61578768491745, 0.2537577748298645, 0.7635567784309387, -0.2448962777853012, -0.828743577003479, 0.23235760629177094, -1.2524133920669556, 0.30297401547431946, 0.0786610022187233, -0.06449804455041885, -0.406707227230072, -0.0031896785367280245, -0.19675493240356445, -0.6771618723869324, 0.929718017578125, -0.26456522941589355, -0.6677417159080505, 1.0136576890945435, 0.7190391421318054, 0.2459246963262558, 0.4432702660560608, -0.4675717353820801, 1.9437806606292725, 0.6314215064048767, -0.5134113430976868, 0.2519983649253845, 0.563503086566925, 0.8893731236457825, 0.5952334403991699, 0.5260789394378662, -0.9565145373344421, 1.6487349271774292, -0.9583622813224792, -2.8050038814544678, 2.4007911682128906, -0.15117910504341125, -0.6634207367897034, 0.4204919934272766, 0.46922528743743896], [1.3468126058578491, 1.2398196458816528, -0.8461803197860718, -0.6072614789009094, -2.8510453701019287, 1.2235277891159058, 0.7093676924705505, 0.17218413949012756, 0.23841999471187592, 0.8787493109703064, 3.2411246299743652, -0.7381840944290161, 0.12146009504795074, -0.9431403875350952, 1.266751766204834, -1.4352984428405762, -0.4914001226425171, 0.19042953848838806, 0.15298248827457428, -0.8258501887321472, 1.2819736003875732, -0.8548014163970947, 0.147400364279747, -0.8955315947532654, 0.9686111211776733, -0.7354399561882019, 0.4525608420372009, -0.10874764621257782, 0.44059309363365173, 1.1977952718734741, -1.5002940893173218, 0.2257232666015625, 0.9802044630050659, 0.11196025460958481, -0.3472293019294739, -1.007625937461853, -0.5744559168815613, 0.3637193441390991, 1.5909348726272583, -0.3522202968597412, 1.77468740940094, -0.228739932179451, -0.09717024117708206, 0.02647402323782444, -0.8389022350311279, 0.21648666262626648, 3.9192919731140137, 1.7969564199447632, -0.5707775354385376, -0.7204480767250061, -0.5471813082695007, 1.1146588325500488, 1.3569424152374268, 0.5530954599380493, -0.8366857767105103, 0.4565785229206085, -0.40315642952919006, 1.2515660524368286, -1.5873678922653198, -0.322740763425827, 1.3573243618011475, -0.4433263838291168, -2.195619583129883, 2.0355169773101807, -0.4236453175544739, -0.6278564929962158, 0.009071202017366886, 0.8941611647605896, -0.8008943200111389, -1.0605067014694214, 0.9427491426467896, -0.13297219574451447, -0.19828815758228302, 1.8078193664550781, -0.27100053429603577, -0.5962951183319092, 1.1819170713424683, -0.5839314460754395, 1.1283271312713623, -0.2598036527633667, -0.8337517380714417, -0.5639448165893555, 2.1185946464538574, -1.7601574659347534, 1.368768334388733, 1.3443020582199097, -0.8564253449440002, -0.8335898518562317, 0.9171481728553772, -0.16101475059986115, -0.25504881143569946, 0.9405248165130615, -1.025184988975525, -0.5458957552909851, -1.7888120412826538, 1.3585269451141357, -0.27276870608329773, -0.8948217034339905, -2.2410542964935303, -1.2455464601516724, -0.22041353583335876, -1.0604010820388794, -1.2283633947372437, -0.4283161461353302, 0.3335425853729248, 1.21673583984375, 0.25828734040260315, 0.42835646867752075, 1.0204100608825684, -1.218826174736023, -0.5409850478172302, -0.20783814787864685, 1.4918906688690186, -0.650858998298645, -1.2790155410766602, 0.4786207377910614, 0.6325374245643616, 0.668286144733429, 0.1841641217470169, 0.40343132615089417, -0.7550755143165588, -0.882949709892273, 1.4449052810668945, -1.0980334281921387, -1.6084895133972168, 0.6707239151000977, -0.1913505643606186, -0.4800227880477905], [-0.6989756226539612, -0.6089612245559692, 0.0234081968665123, -0.6980553269386292, -0.3908766806125641, 0.7657408714294434, 0.26573604345321655, -3.262014389038086, -0.13038645684719086, 0.19909009337425232, 0.2070556879043579, 1.4158353805541992, 1.004433274269104, 2.244900703430176, 1.0613341331481934, -0.762236475944519, -0.009566393680870533, -0.03233285993337631, 1.8910019397735596, -1.5391552448272705, 1.3649659156799316, 3.0195086002349854, 0.11197452247142792, -0.017299802973866463, -0.3411200940608978, 1.140129804611206, -2.1474013328552246, 1.6002918481826782, -0.24752677977085114, 0.7730295062065125, -0.20475409924983978, 0.048982806503772736, -0.7006978392601013, -1.1003457307815552, -1.6612101793289185, -1.2765517234802246, -0.48576822876930237, 0.024853259325027466, -0.12131395936012268, -1.2900229692459106, -0.0881769135594368, -0.6701353788375854, -1.6459274291992188, 0.1612785905599594, -2.0916242599487305, 2.0978689193725586, -0.41917213797569275, 0.6391353011131287, 0.13017193973064423, -0.09674207121133804, -0.5087329745292664, 0.7405814528465271, -0.2114412784576416, 0.054056305438280106, -1.3697980642318726, 0.7354523539543152, -1.0919759273529053, -0.47660690546035767, -0.22751063108444214, 0.06405878812074661, 0.7307100892066956, 0.7276098132133484, -0.3293766379356384, 1.467738151550293, 1.1196922063827515, 0.1052711084485054, -1.9022670984268188, -0.2820841670036316, -0.6347451210021973, -0.8868910670280457, 2.0628318786621094, 0.6472142338752747, -0.5868077874183655, -0.2638034224510193, 1.7285808324813843, 0.6313163042068481, -0.7258181571960449, -1.2103582620620728, -0.7834433913230896, 0.5780724287033081, -0.28919368982315063, -0.006470491643995047, -0.9451074600219727, 0.8441827893257141, -0.577683687210083, -1.5451606512069702, -0.5946456789970398, -0.535986602306366, 1.0605289936065674, 0.3149532973766327, 0.39023563265800476, -0.4054993689060211, 0.9077261686325073, -0.3152689039707184, 0.27882033586502075, -0.7483629584312439, -1.169482946395874, 0.3556811511516571, 0.7958321571350098, -1.2424331903457642, 0.4155846834182739, 0.2999195158481598, -0.5163248181343079, 1.6595045328140259, -1.0534354448318481, 0.17078238725662231, 1.24300217628479, -0.5937328338623047, 0.3880726397037506, 0.9816586971282959, 1.7852389812469482, 0.01848624460399151, -1.4346694946289062, -0.3851650059223175, 1.501244306564331, 1.1263017654418945, 0.680270791053772, -0.8592959642410278, -0.724274754524231, 0.11113779246807098, 1.4231071472167969, 1.4511542320251465, 0.9984778165817261, -2.764634847640991, -2.055459976196289, -1.7300515174865723, 0.08168909698724747, -0.43491020798683167], [1.6548995971679688, 0.05092289298772812, 0.21300768852233887, 1.7989833354949951, -0.9818500876426697, -0.45209479331970215, -0.44559451937675476, -1.2336512804031372, -0.02487199567258358, 1.0927813053131104, 1.064971923828125, 2.6160507202148438, 0.04148393124341965, 0.8226006031036377, -0.806178092956543, 0.4008323848247528, -0.39143553376197815, 0.6634701490402222, 0.7742138504981995, -0.038501810282468796, -0.2702837586402893, 1.5169917345046997, 0.14704637229442596, 1.4441651105880737, -0.6182429790496826, 0.3515569865703583, -0.9557662606239319, -2.0579068660736084, -0.08171635121107101, 1.6265040636062622, -2.0397136211395264, -0.6911991238594055, 1.6004072427749634, -1.5672557353973389, -0.4220239818096161, -0.001145820482634008, 0.7288466691970825, 1.3969968557357788, 1.0598502159118652, 0.4535851776599884, 0.11060585081577301, -1.2234697341918945, -0.25956329703330994, -2.4476561546325684, -0.519229531288147, -0.5222124457359314, 1.1214752197265625, 0.5109778046607971, -0.24133798480033875, 1.3452966213226318, 0.5179975628852844, -0.532638669013977, 0.47853511571884155, -1.1648683547973633, 0.8674281239509583, -0.04529562592506409, -1.3794196844100952, 1.9340471029281616, 0.1688825488090515, 1.052754521369934, 0.48907363414764404, 0.9783180356025696, 0.9684199094772339, 0.8736833333969116, 0.27077168226242065, 0.1507340669631958, 0.10832548141479492, 1.2048600912094116, -0.6197642087936401, -0.24743494391441345, -1.4586546421051025, 1.0790083408355713, -0.39345982670783997, -0.8821171522140503, 0.16549909114837646, -0.2372031807899475, 0.3590046167373657, -0.2227047085762024, 0.9978206753730774, -1.2491041421890259, -0.5674616098403931, 1.4564746618270874, -2.3098697662353516, 0.609622061252594, -0.8325169682502747, 0.8311412334442139, 0.5340386629104614, -0.36848220229148865, -1.0779297351837158, 0.6135368347167969, -1.382136344909668, 0.693406879901886, 1.0435950756072998, -0.5179802775382996, 1.3412179946899414, 0.7575931549072266, -0.7933996915817261, 0.9467645883560181, -0.4871880114078522, -0.8917332291603088, -0.48374900221824646, -0.8474253416061401, -0.4924236834049225, -0.13561375439167023, 0.5908262133598328, 0.8141065835952759, 0.08821883052587509, -0.4259239137172699, 1.0558782815933228, 0.5505627989768982, 1.2048224210739136, 0.20248599350452423, -0.4952828586101532, 0.43812695145606995, -0.006717176176607609, -2.2634918689727783, -0.020362766459584236, -0.5089556574821472, 1.2105909585952759, 1.1728764772415161, -1.1654661893844604, -0.46953386068344116, -0.24892596900463104, 0.6161218881607056, -0.535045325756073, 0.17261403799057007, -0.8310906291007996, 1.493178367614746], [-0.5632355809211731, 1.3873368501663208, 1.6563035249710083, -0.900322437286377, -0.49589699506759644, -2.4394588470458984, 0.19802404940128326, 0.7131811380386353, 0.49243342876434326, 1.5713192224502563, 0.9584794044494629, -0.8236250877380371, 0.006321652792394161, -1.0857937335968018, -0.2159905880689621, 0.13670209050178528, -1.8844693899154663, -1.1851458549499512, -0.8982839584350586, -0.21907161176204681, -0.01544396672397852, 0.4937497079372406, 1.6915847063064575, -0.7096880078315735, 1.934309720993042, -0.7576727867126465, -1.151611566543579, 0.7337929010391235, 1.8626618385314941, -3.784834623336792, -1.9453834295272827, 0.8255338668823242, 2.1769003868103027, 0.6907336115837097, -0.9672653675079346, -0.2559116780757904, -1.070647120475769, -0.8690212965011597, -0.06225958094000816, -0.588786244392395, 0.26758235692977905, -0.2679366171360016, 1.2343149185180664, 0.9382632970809937, 1.2036769390106201, -0.5303970575332642, -0.7508167028427124, 0.27132654190063477, 0.12605035305023193, -0.29535216093063354, 0.7912769317626953, -0.038343094289302826, 0.5178690552711487, -0.2436906099319458, -0.16525065898895264, -0.4116605222225189, 0.8607872128486633, 0.24148300290107727, -0.19173628091812134, -0.42913761734962463, -0.5457761883735657, -1.1006500720977783, -0.1550423800945282, 1.029454231262207, -0.3502510190010071, 0.0349162332713604, -1.2052040100097656, -1.1911567449569702, -1.3162676095962524, 0.09923974424600601, 1.2263246774673462, -0.039524514228105545, 1.074379801750183, -0.7875345945358276, 0.273578941822052, -0.5943531394004822, 0.20288710296154022, -1.019770622253418, 1.4007779359817505, -1.0967704057693481, -0.36660799384117126, 0.3850555121898651, -1.7287442684173584, 0.904757022857666, 0.3768339455127716, 0.2527112662792206, -0.6739229559898376, 0.3798764944076538, 1.9653589725494385, 0.6476853489875793, 0.46062883734703064, -0.6035140156745911, -0.014671873301267624, 0.24533618986606598, -0.5466693639755249, -0.8057200312614441, -0.6258488893508911, -0.8241367340087891, -0.0534442700445652, 0.06471430510282516, 1.5634346008300781, 0.5035514235496521, -0.6930485963821411, 0.5236853957176208, 0.6076295375823975, -1.4615155458450317, 0.06756872683763504, 1.020047664642334, -0.24881011247634888, -0.14494358003139496, -0.16402402520179749, -0.04861733689904213, -0.03407468646764755, 0.5101407170295715, 0.7206521034240723, 0.7355779409408569, -0.527150571346283, 1.328454613685608, 1.5443629026412964, 1.0160107612609863, -0.8605939149856567, -0.6072378754615784, -0.19687063992023468, 0.49285170435905457, -0.08706964552402496, -2.49078369140625, -2.007993459701538, -0.40983903408050537], [-0.7424718737602234, -1.2189207077026367, 1.0102310180664062, -0.021703006699681282, 0.5583935379981995, -0.6871292591094971, 0.6551117897033691, 1.2664638757705688, 0.6819502115249634, 1.658722162246704, 0.9391435980796814, 0.3477816581726074, -1.832047939300537, -0.1371670961380005, -2.503960132598877, 0.571535050868988, 0.8937636613845825, -0.29219093918800354, 2.0297985076904297, -1.4180692434310913, -1.8302979469299316, 1.4005153179168701, -1.6392585039138794, -1.7082269191741943, 0.3429115414619446, -0.8547420501708984, -0.9672390818595886, 1.1235976219177246, -0.5312827825546265, 0.6837803721427917, 1.050561785697937, 0.6727798581123352, 0.5639749765396118, 1.3273015022277832, -0.3116634488105774, 0.01676183007657528, 0.17671339213848114, -0.12771080434322357, 0.624853789806366, 1.5980340242385864, -2.9518167972564697, 0.2142687290906906, 0.9826899766921997, 1.7771453857421875, 0.14351670444011688, -0.07899907231330872, -0.5087264776229858, -1.7539931535720825, -0.7806715965270996, -1.1355268955230713, 0.6617454886436462, -1.0682929754257202, -1.7984546422958374, -0.07686896622180939, -0.1339195966720581, -1.9680370092391968, -0.8330596685409546, -0.5951038599014282, 1.431915521621704, 0.4157077670097351, 0.45962584018707275, 0.7508911490440369, -0.17607976496219635, 0.45782917737960815, -0.8185219168663025, -0.8270999193191528, -1.4841828346252441, 1.2375563383102417, 0.5035213232040405, 0.9607270956039429, 0.08641303330659866, -0.34405478835105896, -1.4983396530151367, 2.134481430053711, 0.7022513747215271, 1.003947377204895, -0.033367402851581573, 0.639444887638092, -0.1057409942150116, 1.0230541229248047, 0.6302642226219177, 0.3208603262901306, -0.33513468503952026, 0.6697139739990234, -2.9017248153686523, 0.6049721240997314, -1.3026955127716064, -0.48508748412132263, 1.833580493927002, -0.3474310040473938, -0.8003436326980591, -0.3755069673061371, -1.2515827417373657, 2.0390536785125732, -0.11660704761743546, 0.6990674138069153, -2.2903268337249756, 0.8966825604438782, -1.347080945968628, -1.0348833799362183, 0.9621413350105286, -0.7324299812316895, 1.9269380569458008, -1.0928428173065186, -0.585071861743927, -0.5120554566383362, -0.3690527379512787, -0.19126203656196594, 0.7789895534515381, -1.7956904172897339, 1.410448431968689, 1.3217781782150269, -1.1772786378860474, 0.9196853041648865, -0.9607906937599182, -0.7600651383399963, -0.1852118968963623, -0.8624851703643799, -0.7076956033706665, 1.0662282705307007, -0.06040225923061371, 2.109149932861328, 1.8719432353973389, 1.881333351135254, 1.464568853378296, -0.8294231295585632, -0.3321828842163086, 0.1326683610677719], [1.3945369720458984, -2.1887435913085938, 0.05234510079026222, 1.5062892436981201, -1.2162890434265137, 0.33568525314331055, -0.3250705599784851, 0.24966338276863098, -1.0712693929672241, 0.2848719358444214, 1.1571764945983887, 0.276867538690567, 1.4668744802474976, -0.5597755312919617, -1.890359878540039, 0.11312025040388107, 1.8376781940460205, -1.4589614868164062, 0.914429247379303, -0.45627620816230774, -0.5566643476486206, -0.683754026889801, 0.23749971389770508, -1.7455142736434937, -0.33903178572654724, 0.07636924833059311, -1.8719202280044556, -1.2821491956710815, 0.927047848701477, -1.2736374139785767, -0.0492459312081337, -1.872238278388977, -1.5982950925827026, -0.033062104135751724, 0.17357264459133148, 0.8401107788085938, 0.315463125705719, 1.772408366203308, -1.3495168685913086, 1.2415978908538818, 1.0355292558670044, 0.041669026017189026, 0.535368025302887, 0.3211529850959778, 0.5671623349189758, 0.17217373847961426, 0.2985934317111969, -0.027692489326000214, -1.4879040718078613, 0.9135210514068604, -0.16283127665519714, 0.8629482984542847, 0.033769216388463974, 0.06051235273480415, 0.27840158343315125, 0.43124833703041077, 0.5104841589927673, -0.14416734874248505, 0.30892476439476013, -0.04348626360297203, -0.1459350883960724, -0.4464893937110901, 0.0272369347512722, 1.4457581043243408, -0.5730645656585693, -1.0027432441711426, 0.7895936369895935, 0.5767456889152527, -1.9835715293884277, 0.5004451274871826, 0.8632112741470337, 1.3323253393173218, -0.737375795841217, 1.157801866531372, -0.8212834000587463, -0.19563235342502594, -1.901475191116333, -0.4217582941055298, 0.9816564321517944, -1.479397177696228, 1.1614608764648438, -0.05338146165013313, -0.11433286219835281, 2.8312227725982666, 1.2597672939300537, 2.0587470531463623, 0.0019865341018885374, -0.440550833940506, 0.2688768804073334, 1.4584226608276367, -0.349351167678833, 0.3131820261478424, 0.6567233800888062, -1.3477224111557007, -0.023386653512716293, -0.10239271819591522, 0.784195065498352, -0.10257235169410706, -0.4073459506034851, 1.643672227859497, 0.2002735435962677, 0.20711347460746765, 0.23344570398330688, 0.2877582013607025, -1.0257357358932495, 0.41943392157554626, -0.49459710717201233, -0.7137543559074402, -2.7868804931640625, -1.697707176208496, -0.9444709420204163, -0.04469746723771095, 0.20901815593242645, 0.8990409970283508, 0.37585651874542236, 0.28758710622787476, -1.805021047592163, 0.5164909958839417, -0.8079023361206055, 1.9197683334350586, 0.6280403137207031, -1.2155206203460693, -0.5834160447120667, -0.5227894186973572, 1.314574956893921, 0.16879040002822876, -1.7786537408828735, 0.18329556286334991], [-0.4966244399547577, 1.2313816547393799, -0.2668014466762543, 0.6048604846000671, 0.06277398020029068, 0.42733269929885864, -0.08788275718688965, 0.49810945987701416, 1.3166922330856323, 0.061130572110414505, -0.31752198934555054, 1.8993703126907349, -2.0343661308288574, 1.4929218292236328, 1.0334086418151855, 0.4534750282764435, -0.7577546834945679, -0.9207958579063416, 0.54932701587677, -0.6315317749977112, 0.689294159412384, -0.06649930030107498, -0.3867926299571991, 1.0053684711456299, 0.7887251377105713, 0.1172695979475975, -1.021459698677063, 0.9041957259178162, -1.3456425666809082, 0.22166398167610168, 0.12300646305084229, 0.5390583276748657, 0.8014519810676575, -2.123492479324341, -0.11468736827373505, 0.15153098106384277, 0.7875960469245911, -1.0029059648513794, -0.1042182520031929, 1.0467031002044678, -2.4547653198242188, -0.6538052558898926, -0.2987419366836548, 1.0239503383636475, 1.1910592317581177, -1.5908957719802856, -1.6066313982009888, -1.5466076135635376, 1.559889316558838, -0.8933194279670715, -0.29235294461250305, 1.9552011489868164, 0.7001263499259949, 0.7509555816650391, 1.0982153415679932, 1.2376203536987305, -0.4220498502254486, -0.5548956394195557, -1.324030876159668, -0.43655431270599365, -0.6807094812393188, 0.5006273984909058, -0.05953628569841385, -1.5166027545928955, 0.205842524766922, -0.17399947345256805, -0.5955467820167542, 1.5185104608535767, -1.4000071287155151, -0.23812861740589142, -0.13493512570858002, 0.2105790376663208, -1.347205638885498, 1.3172003030776978, 0.35163912177085876, -1.3043832778930664, -0.8459954261779785, 0.16731494665145874, 0.5733267068862915, 0.39658722281455994, 0.19683107733726501, 0.7305116057395935, -0.6485216617584229, 1.2457787990570068, 1.1375524997711182, 1.0702468156814575, 0.17949038743972778, 0.4980950355529785, -0.17337098717689514, 0.10187989473342896, 0.6197662949562073, -0.4018521010875702, 0.45491722226142883, -0.36596357822418213, 0.42282986640930176, 1.2103244066238403, 1.176345705986023, 1.6268103122711182, 0.32917264103889465, 0.5071425437927246, 2.506303548812866, -0.2875687777996063, 0.526745080947876, -0.7376765012741089, 0.25610002875328064, 0.48473188281059265, 0.6260219216346741, 0.45264729857444763, 1.531980037689209, -1.9734547138214111, 0.7866997122764587, 0.7004535794258118, -0.8448794484138489, 0.03626934811472893, 0.6085920929908752, -0.893048107624054, -1.2910512685775757, 0.913415789604187, -0.26455461978912354, 0.20452111959457397, 0.4485395550727844, -0.33735185861587524, 0.37849998474121094, 0.6644996404647827, -0.5090625882148743, 0.3277113139629364, 0.9109209179878235, -0.6509144306182861], [-0.8622997999191284, -0.28894248604774475, 0.4359614849090576, 0.021205240860581398, 0.12671354413032532, 1.5253835916519165, -0.13679493963718414, -1.1319619417190552, -1.2587039470672607, -1.5489060878753662, 0.9239091873168945, 0.9173169136047363, -0.4679183065891266, 0.9546884298324585, -0.16825313866138458, -0.8558207154273987, 0.3187078833580017, 2.579331159591675, -1.3293901681900024, 1.9373157024383545, -0.3650056719779968, 0.3275472819805145, -0.8506484031677246, 0.6439589262008667, 0.6084907650947571, 1.3652193546295166, -1.9098089933395386, 0.34830528497695923, -0.7113310098648071, 0.35139378905296326, -0.8710262775421143, 0.1350686401128769, 0.3435118496417999, 0.4394950270652771, 1.4697099924087524, 0.6106778979301453, 1.1114505529403687, -1.2655742168426514, -0.6855363249778748, 0.1592155247926712, -1.6626899242401123, -0.7620053887367249, -1.0520919561386108, -1.8537813425064087, -1.1964325904846191, -0.029976943507790565, -0.8385424613952637, 1.7752307653427124, 0.8027554154396057, -0.3372238576412201, -0.012204539962112904, -0.25467124581336975, -0.889041006565094, -1.3287030458450317, -1.0764377117156982, 0.5906882882118225, 1.6937216520309448, 0.23586268723011017, 0.6884283423423767, 0.45325517654418945, 1.4733481407165527, -0.034521374851465225, -1.4619836807250977, -1.268178105354309, -0.633189857006073, -1.431470513343811, 0.839670717716217, 0.5528461933135986, -1.373809814453125, 0.7205171585083008, 0.8063639402389526, 0.11253715306520462, -0.45798560976982117, -1.307938575744629, -0.8779356479644775, -0.2497999370098114, 0.9346945285797119, 0.4788982570171356, -0.3617997467517853, -0.11315979808568954, 0.4724235236644745, 0.0004920032224617898, -0.10234405845403671, 0.9032514095306396, 1.2068705558776855, 0.8127568364143372, 0.7952205538749695, -1.4628241062164307, 0.6922202706336975, 1.3046839237213135, -0.6363561153411865, -2.178997278213501, -0.8022419810295105, 0.771980345249176, 0.6655300259590149, 0.2197171151638031, -1.013854742050171, -0.8101763725280762, 0.5296400785446167, 2.3126699924468994, -0.6280056834220886, 0.9624603390693665, 1.2841062545776367, 0.8174829483032227, -1.4888834953308105, 0.6589670181274414, -0.012415238656103611, -0.588050365447998, 0.45201611518859863, -0.09706784784793854, -1.5628182888031006, -1.825595736503601, 1.8139348030090332, 1.115209937095642, -0.8297451138496399, 0.4735676646232605, -0.6620298624038696, -1.6222875118255615, 0.5243555307388306, 0.24453701078891754, -1.5546504259109497, -0.7256485223770142, 0.09430896490812302, -0.6529589891433716, 0.18041610717773438, 0.3832359313964844, -1.1890337467193604, 0.7683270573616028], [0.7565594911575317, -0.2842142879962921, -0.8153767585754395, 0.7062024474143982, -0.1407870352268219, -1.6211754083633423, 0.04181094840168953, -0.024200741201639175, -0.5699110627174377, 0.8400667905807495, 0.8568781018257141, 1.0137537717819214, 0.10657360404729843, -1.2597177028656006, 0.6957716345787048, 0.8723849058151245, 0.43731310963630676, -0.9235053658485413, -0.20749621093273163, 0.5771776437759399, -0.6272879242897034, 0.1573866307735443, -0.9691017270088196, 0.5228204131126404, 0.33929136395454407, 0.1055503860116005, -0.46769481897354126, 1.0021698474884033, 0.4227342903614044, 2.7646422386169434, 2.2100183963775635, 0.3883391320705414, 0.11304788291454315, -0.4687776267528534, -0.09322092682123184, 0.16977404057979584, 0.13847576081752777, -0.945061981678009, 0.30396321415901184, 1.9629589319229126, 0.8323060870170593, -0.4724487364292145, 0.4890528917312622, -0.3143205940723419, 1.6471184492111206, -0.41723647713661194, 1.1079589128494263, -1.5556787252426147, -0.1865662932395935, -1.267609715461731, -0.3961016535758972, -1.5703731775283813, 0.7472653985023499, -0.27126601338386536, -1.03293776512146, -1.027147889137268, 0.3748215138912201, 1.198794960975647, -0.3508644998073578, 1.6967021226882935, 0.09976844489574432, -0.7786556482315063, 0.3228370249271393, 0.6728213429450989, -0.2035776674747467, -0.9995637536048889, -0.501484751701355, -0.11330914497375488, -1.7765358686447144, -1.6152622699737549, 0.6334791779518127, 0.8015154004096985, 2.2484612464904785, 1.653214454650879, 0.774218738079071, -0.3769580125808716, 0.35327059030532837, -0.8906023502349854, 0.5397441387176514, 0.30950483679771423, 0.6646400094032288, 1.1101816892623901, -0.3142932951450348, 0.21637092530727386, -0.1475284993648529, 1.077648639678955, -0.5567023754119873, -0.2105904072523117, 0.38861870765686035, 0.04666053131222725, 1.7077122926712036, 0.07224210351705551, -0.826228678226471, 0.5678837895393372, -0.3435814380645752, 0.567779004573822, 0.1821478307247162, -0.15004447102546692, 0.9502541422843933, 0.8161904215812683, 0.15351226925849915, -1.7952932119369507, -0.2212299406528473, 0.14886675775051117, -1.231119990348816, -0.9274924397468567, -0.2820729613304138, -1.7886183261871338, 0.011465666815638542, 0.23748326301574707, -0.2678654193878174, 0.6040937304496765, -0.3988804817199707, 0.5910070538520813, 1.41261625289917, 0.2904631793498993, 1.019715666770935, -0.42408955097198486, 0.7622535228729248, -0.22174187004566193, -1.5977606773376465, 0.5514046549797058, 0.23446890711784363, -1.1904088258743286, -0.6122871041297913, 0.27604520320892334, -1.3711291551589966, 0.622901976108551], [0.8234130144119263, -0.45438286662101746, 0.029662195593118668, 0.33278435468673706, 1.2291139364242554, -0.37249690294265747, -1.1425329446792603, 1.1485577821731567, -0.08559730648994446, 0.7527541518211365, -0.12157316505908966, 0.20399636030197144, -0.701350212097168, -0.11219246685504913, -0.2139553725719452, 0.22188915312290192, 1.0217722654342651, -0.6118366718292236, 0.9258009791374207, 1.7263582944869995, -1.4250184297561646, -0.20353932678699493, 0.09506019949913025, -1.2100439071655273, 0.4768548309803009, -0.28727948665618896, 1.8286372423171997, -0.06963665783405304, -1.3357231616973877, 0.34710511565208435, -0.8795841932296753, 0.8727853298187256, -0.9131662249565125, -0.9375152587890625, -1.1034985780715942, -1.7978708744049072, 0.4171693027019501, 0.45526042580604553, -0.6800351738929749, 0.08850879967212677, 0.5279372930526733, 0.5875232815742493, -0.3606540858745575, 1.4748457670211792, -0.5989969968795776, -0.54029780626297, 2.917659044265747, 0.952104926109314, -1.2472792863845825, -0.525526762008667, 1.0520548820495605, 0.4289451539516449, -0.2916528582572937, -1.805116891860962, 1.6124951839447021, -3.24283504486084, 0.021313214674592018, 0.08529645949602127, -1.98397696018219, -0.28731393814086914, 0.16497382521629333, 0.2187376618385315, 1.84028959274292, 0.2906523048877716, -0.04874328151345253, 0.3980984389781952, -0.559052050113678, -1.1424223184585571, 1.2992092370986938, -1.7535028457641602, -1.3840034008026123, 0.2357650101184845, 0.754631757736206, 1.5123305320739746, 0.5486130118370056, 1.385111927986145, -0.24362847208976746, -1.6014431715011597, 0.30476754903793335, 0.41046881675720215, 0.7277231216430664, 0.062051594257354736, -1.612120509147644, -0.40359851717948914, -0.9891401529312134, -0.0911535993218422, 0.9421119093894958, -1.074264645576477, 0.11011788994073868, 0.9335228800773621, 0.27386337518692017, -0.6655332446098328, -1.8918635845184326, 0.8801284432411194, 0.9216731190681458, -1.544969081878662, -0.3422028720378876, 0.1755581796169281, 0.08533183485269547, -0.13274843990802765, -0.45594868063926697, 0.478941947221756, -2.192838191986084, -0.31386798620224, -1.1803150177001953, -0.42789748311042786, 1.0601603984832764, -1.0027046203613281, -0.5249610543251038, -0.09650009870529175, -2.4666409492492676, 0.4840660095214844, -1.3849644660949707, 0.04067175090312958, 0.3862036466598511, 2.5837252140045166, 0.6001251339912415, -0.6819326281547546, 1.6772416830062866, 1.5989100933074951, 0.020870793610811234, -0.6309933066368103, -0.2602255940437317, -0.3790099620819092, 1.4187376499176025, -0.6622565984725952, 2.2648847103118896, 0.4031396508216858], [-1.1945874691009521, 0.4558647871017456, 0.9383184313774109, 1.212079405784607, -0.28974902629852295, 1.5669009685516357, 0.2285921573638916, -1.4099980592727661, -0.2693069577217102, 0.10400794446468353, -0.07222166657447815, 1.4977749586105347, -0.5880477428436279, -0.15689627826213837, -0.667439341545105, 2.0244245529174805, -0.20697849988937378, 0.17300641536712646, 0.7556427717208862, 0.6582723259925842, 0.9434027075767517, -0.2649911046028137, 0.605088472366333, 0.1439952552318573, -0.5105141997337341, -0.28130072355270386, 1.1042755842208862, -0.3748871982097626, -0.23001284897327423, 0.7186799645423889, -1.057750940322876, -0.497955858707428, -0.43436598777770996, 0.010255459696054459, 0.36472851037979126, -1.4192944765090942, 0.14186538755893707, -1.0029546022415161, 1.0493091344833374, 1.3483484983444214, -0.2822191119194031, 1.3099333047866821, 0.4920402765274048, 0.5509594678878784, 0.32638323307037354, 0.5303411483764648, 0.1815379112958908, -1.0262908935546875, -0.3961615264415741, -1.733276128768921, 0.904141366481781, -0.6834386587142944, 2.2628719806671143, -0.12155424058437347, 1.4770642518997192, -1.5542072057724, -0.4994603395462036, -1.4385368824005127, 2.1182425022125244, -0.2815050184726715, -1.480025291442871, -0.34606674313545227, -0.5678040385246277, 0.7932085990905762, -0.6945303678512573, 0.5079519152641296, 1.0070222616195679, -1.4751129150390625, -0.3134469985961914, -0.2527419626712799, -2.0143141746520996, -1.376089096069336, -1.974181890487671, -1.6810569763183594, 0.6167836785316467, 0.6950915455818176, 0.5009627342224121, -1.07069730758667, 0.8685047626495361, 1.5380926132202148, 1.268444538116455, -0.03067629784345627, 0.7004007697105408, -0.47838112711906433, -1.2214518785476685, 0.21222688257694244, 0.530742883682251, 1.2348439693450928, 0.18234938383102417, -0.6468806862831116, 0.8036013841629028, 0.3978578448295593, -0.5977055430412292, 0.20074769854545593, -0.7522698640823364, 0.4217672049999237, 0.28490114212036133, 0.2009502798318863, -0.2971593141555786, -0.14243347942829132, 1.063400149345398, 0.7209349870681763, -1.6237396001815796, -1.5509576797485352, -0.356570303440094, 1.2846921682357788, -0.9160302877426147, 1.6883113384246826, 1.772749423980713, -0.6681764125823975, -0.04766669124364853, 1.028183937072754, -0.23457689583301544, 0.467874675989151, 0.9137417674064636, -1.7421362400054932, 0.5139992833137512, -0.7239978909492493, -1.448677897453308, 1.687333345413208, -0.6004539132118225, -0.6483697295188904, 2.6918888092041016, 0.30367034673690796, 0.648576021194458, -0.8575601577758789, -0.8455260992050171, -0.21002189815044403], [-1.5857831239700317, 0.6191878318786621, -1.28817880153656, -0.5002008676528931, -0.22529470920562744, -1.3406826257705688, 0.8523648977279663, -0.8433981537818909, 0.9303467869758606, -1.2028948068618774, 0.709297239780426, 0.19952544569969177, 1.2871406078338623, -0.18502403795719147, 0.5597273707389832, -0.8075486421585083, 0.9509680271148682, 1.722025752067566, 1.8724689483642578, 1.9858533143997192, -1.0973303318023682, -0.6884850263595581, -0.3010598123073578, 0.22008781135082245, -0.7930852770805359, 0.16909897327423096, -0.7880614995956421, -0.7760675549507141, 0.0869261845946312, -0.07564068585634232, -0.5835213661193848, -0.5971148014068604, -0.0031460868194699287, 0.5957187414169312, 2.0583930015563965, 0.4656514823436737, 2.24589467048645, 1.598362684249878, 0.4466097950935364, -1.205599308013916, -0.04482110217213631, -0.017014699056744576, 1.1050009727478027, 0.7091229557991028, -0.6781530976295471, 1.1391713619232178, -0.972657322883606, 1.4188320636749268, 1.9082043170928955, -0.786516010761261, -1.3284480571746826, 0.2235911786556244, -0.0757441595196724, 0.0810021236538887, -0.850917637348175, -0.1915479451417923, 1.5671764612197876, 0.2619302570819855, -0.36642399430274963, 0.1577862650156021, -1.621389389038086, 0.17841677367687225, 1.7388654947280884, 0.05399373173713684, 0.34080901741981506, -0.20424893498420715, 0.5624572038650513, 1.5335655212402344, 0.37377217411994934, 2.0156517028808594, 0.5408284664154053, -0.6263664364814758, -0.41381651163101196, -1.043474555015564, 0.14283086359500885, 1.1305166482925415, -0.05838265269994736, 0.9340142607688904, 1.0027779340744019, 1.4287166595458984, -0.552708625793457, 1.1125274896621704, 0.5718362331390381, -0.5459086298942566, 1.603287696838379, 0.7304250001907349, -0.8506794571876526, 0.4993237257003784, 0.061021946370601654, 2.1839489936828613, -0.670260488986969, -1.6799062490463257, 2.1215200424194336, 1.055606484413147, 0.39665499329566956, -0.743718147277832, 0.15854361653327942, -0.7515149116516113, -0.6631274819374084, -0.16467516124248505, -0.020262449979782104, 2.2592923641204834, 1.429681420326233, 0.5516888499259949, 0.41037294268608093, 0.6744397878646851, -0.031419891864061356, -1.8019579648971558, 1.3296648263931274, 1.2999908924102783, 0.002032372634857893, 0.7405509948730469, 0.6970211267471313, 0.7566086649894714, -1.468945860862732, -1.3143250942230225, -1.9884207248687744, 0.33486250042915344, -0.34446045756340027, 0.15139363706111908, -0.5952470302581787, -0.017044642940163612, 2.0817043781280518, -0.20747621357440948, -1.130898118019104, -1.3242956399917603, 0.47397738695144653, -0.30208003520965576], [-1.5755468606948853, -0.9428077340126038, 1.1392385959625244, 0.06089157983660698, 0.17002959549427032, -2.123746156692505, -0.3676760196685791, -0.03207795321941376, -0.9083617329597473, -1.2127840518951416, -0.28831562399864197, 1.1838897466659546, -0.2435285896062851, -1.8186345100402832, 0.209671750664711, 0.8332332372665405, -1.41754150390625, 0.5321266651153564, -0.010736863128840923, 0.5806208848953247, -0.5555709004402161, -0.11440341919660568, -0.19031567871570587, 0.2466910183429718, 0.08195528388023376, -0.3207704722881317, -0.8352629542350769, -1.8260830640792847, 3.093604326248169, 0.27998143434524536, -0.003372374689206481, -1.1731356382369995, 0.0009064293699339032, 0.6008813381195068, 0.5452737212181091, -0.9109047651290894, -0.671276867389679, 0.0682809129357338, 0.7137423157691956, 0.46047982573509216, -1.456971526145935, 0.6499326229095459, 0.834935188293457, 0.3100498616695404, 0.5388136506080627, 0.9835342168807983, -0.27651843428611755, -0.116878941655159, 0.2267395257949829, 0.8592209815979004, 1.0510543584823608, 0.5698727369308472, 0.27379027009010315, -0.5751916766166687, -1.077896237373352, 0.3508351743221283, -1.8587828874588013, -1.1705715656280518, 0.2217796891927719, -1.3194692134857178, -0.9248193502426147, 2.9294896125793457, 0.381817489862442, -1.087397575378418, -0.920177161693573, -0.2655741274356842, -0.5813751220703125, 0.18247927725315094, 1.2600780725479126, -1.2352631092071533, -0.062300216406583786, 0.4016275107860565, 0.6508580446243286, -1.564854621887207, -0.7898463606834412, -0.43431153893470764, -0.08660385012626648, 0.6208560466766357, 0.9874556660652161, -0.38403674960136414, -1.1883455514907837, -0.9698981642723083, 0.4157487154006958, 0.08557858318090439, -0.261001318693161, 1.558609962463379, -1.582410454750061, 0.6806780695915222, 1.357163906097412, -1.3596247434616089, 0.36845797300338745, 0.3665068447589874, -0.5396491885185242, -0.05651214346289635, -1.4463623762130737, -0.3786013126373291, 1.9221885204315186, 2.1952574253082275, 1.016908884048462, 0.6602616310119629, 0.80146324634552, 1.400316834449768, 0.6344254612922668, -0.5199176669120789, -1.1751617193222046, -1.2611184120178223, -0.13378803431987762, 0.37588921189308167, -0.9352788329124451, 1.5756686925888062, -0.05593233183026314, 0.15485554933547974, 1.8676421642303467, -1.250511646270752, -0.8290877938270569, -1.44438898563385, 2.0229415893554688, 0.07554257661104202, -0.20148210227489471, 0.9983507990837097, -2.211559295654297, -0.0073060509748756886, 0.7676624655723572, 2.000816822052002, -0.963685929775238, -1.4856377840042114, -0.1695987731218338, -0.25604447722435], [1.5785367488861084, 1.639958381652832, -0.6131051182746887, 1.8151607513427734, 1.203959584236145, -1.0463110208511353, 0.13052591681480408, 0.9949732422828674, 1.8185137510299683, 1.3420051336288452, 0.359113872051239, 1.1965166330337524, 0.27006399631500244, 1.0621556043624878, -0.2142377346754074, -0.929552435874939, 0.7542343139648438, -1.0455601215362549, -0.5149068832397461, -0.3888445198535919, -1.098559856414795, 1.1077852249145508, 0.2457544207572937, 1.3463608026504517, -0.8406965136528015, 0.5301397442817688, -0.34608352184295654, -0.8420020937919617, -0.7230731248855591, 0.46959567070007324, -0.7733767628669739, 2.480560541152954, 1.6753687858581543, 0.7132502794265747, 0.4058551490306854, 0.8935560584068298, -1.4380528926849365, 0.09622196108102798, 0.04001499339938164, -0.1951059252023697, 0.037045758217573166, -0.38226914405822754, 0.3769805431365967, -0.7931156754493713, 0.7442998886108398, 2.0409812927246094, 1.3738960027694702, 0.22556477785110474, -1.6227854490280151, 0.12932904064655304, 1.2243026494979858, -0.6322495341300964, 0.03548334911465645, 1.288041114807129, -1.3347562551498413, 0.8168092966079712, -1.9795677661895752, -0.4199254810810089, -0.0204001534730196, -0.6601029634475708, 0.1843864619731903, -0.3885456621646881, 1.4519286155700684, 0.15240924060344696, 0.6690459251403809, -1.0425231456756592, 0.259880393743515, -2.5560083389282227, -0.913844645023346, 0.8040894269943237, -0.6182993650436401, 0.0523531399667263, -0.1302090734243393, -1.0303747653961182, -2.01867413520813, -0.4286888837814331, -0.4641100764274597, -1.1284234523773193, -1.431792974472046, 0.26959022879600525, 1.3489980697631836, -2.9283764362335205, -0.1837066262960434, -1.9258490800857544, 0.1746651977300644, -0.8306698203086853, 0.984034538269043, -1.3289003372192383, -1.245200753211975, 0.4988853633403778, -0.8611681461334229, 2.6476216316223145, 0.8270875215530396, 1.1467626094818115, 1.7751399278640747, -0.5789220333099365, -0.6015247702598572, -0.05920518562197685, -0.023683827370405197, 1.1278154850006104, -0.3779609203338623, 1.1128720045089722, 0.3147582411766052, -0.6066813468933105, -0.10788961499929428, 0.18286874890327454, 1.4679334163665771, 0.22093422710895538, -0.5834985971450806, 1.0739688873291016, 0.5035768151283264, 1.6121790409088135, 0.08248542994260788, 0.27828314900398254, -0.004346068948507309, -0.22005733847618103, -0.0037791042122989893, -1.1655858755111694, -1.1619020700454712, -0.4974851906299591, -1.009321928024292, -0.16625040769577026, -1.1723287105560303, 0.45229846239089966, -1.0291423797607422, 1.1942458152770996, 0.5191335082054138, -0.40493106842041016], [-1.3377114534378052, 0.25706878304481506, 1.3485842943191528, 0.3754171133041382, 1.4543068408966064, -0.3423083424568176, 0.7952388525009155, -1.8481943607330322, 0.48833566904067993, -0.960098922252655, 0.7166962027549744, -0.7986050844192505, 0.9542317390441895, -1.0401989221572876, 0.4592195153236389, 1.320810317993164, -1.3474177122116089, -1.592621088027954, 0.25923487544059753, 0.07554244250059128, -0.10406024008989334, 0.5059288144111633, -0.46413475275039673, -0.5485216975212097, -2.030606985092163, 0.21031570434570312, -1.2152246236801147, -0.8277778029441833, 0.7310110330581665, -2.0542242527008057, 0.13864341378211975, -0.7695790529251099, 1.631450891494751, 1.1642166376113892, -0.5246835350990295, -0.17324237525463104, -1.1051894426345825, 1.1712599992752075, -0.057612061500549316, -1.1542571783065796, -0.14885610342025757, 1.0070146322250366, 0.6746516227722168, 0.3736543357372284, -0.1276867836713791, 0.5448930263519287, 0.7161319255828857, 0.5587740540504456, 0.05145236849784851, 0.4685536324977875, 1.1867631673812866, 0.3021952509880066, 0.13267222046852112, 0.33141499757766724, -1.317785620689392, -0.35537460446357727, 0.453124076128006, 0.4073197543621063, -0.4345951974391937, 0.0005192619864828885, 0.7165915369987488, 0.327172189950943, 2.530574321746826, -0.2690153121948242, 1.0917176008224487, 0.18448974192142487, 0.6128479838371277, -1.5938208103179932, 0.4894081652164459, 1.6278809309005737, 0.3545002043247223, -1.568317174911499, -1.2159359455108643, -0.2954398989677429, 1.3038190603256226, -0.427724689245224, -0.6524084210395813, -1.0739104747772217, -0.44400519132614136, 0.88075852394104, -0.5661770105361938, 1.4789319038391113, 0.46909624338150024, -0.9881116151809692, 0.8222607970237732, -1.2633403539657593, -1.8499975204467773, 0.7026697993278503, 0.3344527781009674, 0.048074670135974884, -0.24354541301727295, 2.020265579223633, 0.23259004950523376, 2.5409204959869385, 0.13614225387573242, 1.591578722000122, -0.0424920916557312, -0.17536096274852753, -0.13970515131950378, 1.9141076803207397, 0.55314701795578, 0.9042267203330994, 0.599003791809082, -0.5166060328483582, 2.22247576713562, -0.6998989582061768, -0.08723339438438416, 0.6638195514678955, 0.03266027197241783, 0.32393360137939453, -0.19419153034687042, 0.2880812883377075, -0.4963734745979309, 0.09578946232795715, 0.28572043776512146, -0.014994809404015541, -0.0407218299806118, -0.4250965118408203, 0.9596251249313354, 2.545658588409424, -1.8803925514221191, -0.49698349833488464, 0.6945976614952087, -1.2749783992767334, -0.14454522728919983, 1.0123558044433594, -0.36770132184028625, 0.060329701751470566], [0.6887924075126648, -0.9846668839454651, 0.666768491268158, 0.31616050004959106, 0.7040860056877136, 0.8226658701896667, -1.6791208982467651, -0.8587191104888916, 1.0394731760025024, 0.20550915598869324, 1.1138076782226562, -0.6152858734130859, 1.3603044748306274, 1.2214604616165161, 0.753926157951355, -1.5142079591751099, 1.605059027671814, 0.09696948528289795, -0.29074805974960327, -0.38607552647590637, 1.1341543197631836, -0.4781244695186615, -0.030219223350286484, 0.01650373451411724, 0.16519950330257416, -2.486358642578125, 0.5549640655517578, -1.0385246276855469, -0.2924730181694031, 0.8825108408927917, -2.562941312789917, 0.7627599835395813, 0.03967634215950966, 0.3144569396972656, -0.23902744054794312, 0.01613703742623329, -0.9818000197410583, -0.9365770220756531, -0.2592877149581909, -0.24672017991542816, 0.07966995239257812, -1.7622612714767456, -0.0363052636384964, -0.16610948741436005, -0.46996936202049255, 2.1928930282592773, -1.0611869096755981, -0.1278000771999359, -0.022615687921643257, -0.22128057479858398, -0.45706889033317566, 0.4058537781238556, 0.6007781624794006, 1.3737905025482178, -0.18886317312717438, 0.9687783718109131, -0.05441758781671524, 0.24707554280757904, -0.7123536467552185, 0.6448745131492615, 0.41913896799087524, 0.4852672517299652, 0.18605667352676392, 1.1729391813278198, -0.0693809762597084, -1.6597579717636108, 1.013068437576294, 0.8790959715843201, -0.10346178710460663, -0.11946871131658554, -0.5214446783065796, 0.7128600478172302, 0.38445305824279785, -2.5094571113586426, -0.11176467686891556, 0.5761526226997375, -0.9025280475616455, -0.13201121985912323, 0.2123386263847351, -2.1062684059143066, 0.4561033248901367, -0.11367876082658768, 0.42730656266212463, 1.4620054960250854, -3.3337340354919434, 0.5978928804397583, -0.10702349245548248, 0.3614181578159332, -0.5913426876068115, 1.527833342552185, -1.9984411001205444, 1.1376434564590454, 0.07489565759897232, 1.01483952999115, 1.6735508441925049, 0.0770476758480072, -1.2736144065856934, 0.23472042381763458, -1.7312275171279907, 0.17180640995502472, 1.2654529809951782, -1.680354356765747, -0.8734272122383118, -0.5046138167381287, -1.5517674684524536, 0.5513091683387756, -0.3189827799797058, -0.48866304755210876, -0.0813043862581253, 1.612941026687622, -1.5961894989013672, 0.20231162011623383, -0.21451237797737122, -1.05836021900177, -1.5728137493133545, -0.014833931811153889, -0.511293888092041, -0.6254189014434814, 0.3794655203819275, -0.6952444911003113, -0.8004603981971741, 0.2579210102558136, -0.3171803057193756, 0.5095694661140442, 1.0294729471206665, 0.23436714708805084, 0.4265197515487671, 0.08777403086423874], [1.1887648105621338, 0.5410352945327759, -0.9704841375350952, 1.3201912641525269, 1.1666866540908813, 0.4476780891418457, -0.7257848381996155, 1.4934841394424438, -2.2951571941375732, 0.6691559553146362, 1.6036452054977417, -0.48699283599853516, -0.9808226227760315, 0.4680089056491852, 1.966277837753296, 0.8492833971977234, -1.1442912817001343, 0.6745782494544983, 0.28090333938598633, 1.088193655014038, 0.8573925495147705, -0.16767115890979767, -0.685820460319519, -1.5498242378234863, 0.20644868910312653, 0.4303295612335205, -0.4562973976135254, -1.1349059343338013, -1.421937346458435, 0.7411638498306274, -0.24526996910572052, -0.670896053314209, -0.09056127816438675, -0.3613799810409546, 0.5283731818199158, 0.2284502387046814, -1.6080350875854492, 0.9948897957801819, 0.5069026947021484, 0.17954300343990326, 0.19466562569141388, -1.2993550300598145, 1.4577093124389648, -1.3828779458999634, -0.2263706475496292, -0.03568267077207565, -1.1148051023483276, -1.2048468589782715, 0.3739795386791229, 1.1171048879623413, 0.7125608325004578, 0.6398584246635437, -1.5805257558822632, -1.19574773311615, -0.05394084006547928, 0.2484622448682785, -0.22864368557929993, 0.3588763177394867, 0.16956008970737457, -0.8448008894920349, 0.9996290802955627, -1.142316222190857, 0.26813703775405884, 1.9375675916671753, -0.06191238760948181, -0.9015307426452637, 0.6208108067512512, -0.5248680114746094, -1.335205316543579, -1.6210371255874634, -1.0489743947982788, 0.022995127364993095, 2.224052667617798, -0.537312924861908, -1.0247565507888794, -0.6257874369621277, -2.0738275051116943, -0.7273036241531372, -1.333809494972229, -0.023312432691454887, -0.3870801329612732, -0.5266139507293701, 0.20074044167995453, -0.2663068473339081, 0.8952712416648865, 1.4032014608383179, -1.557590365409851, -1.6178526878356934, 1.4300521612167358, -0.18698011338710785, -1.9111895561218262, -0.5206468105316162, 1.534949541091919, -1.6409441232681274, -1.9246926307678223, -1.359149694442749, 0.9230480194091797, 0.04758262634277344, -0.4590280055999756, 1.2024742364883423, -0.8498280048370361, 0.08456544578075409, 0.5511307120323181, -0.36776813864707947, 1.5746608972549438, -0.12355460971593857, -0.2983334958553314, 0.4472513496875763, 0.24374422430992126, -0.4940873682498932, -0.4164245128631592, 2.729400873184204, 0.7803682088851929, 2.262974262237549, -0.480495810508728, -1.6739418506622314, -0.5535395741462708, -0.48546820878982544, 0.11802881956100464, -0.11757536232471466, 0.7616514563560486, 1.5157207250595093, 0.06406471133232117, 0.6681580543518066, 1.6664950847625732, -0.21674352884292603, 0.6407040953636169, 1.3832281827926636], [-0.23688405752182007, 0.6801280379295349, -1.8626128435134888, -0.6871546506881714, -0.7788417935371399, 0.6733531355857849, -1.1900627613067627, 1.1765055656433105, 0.6464734673500061, -0.19419577717781067, -2.0577926635742188, 1.068152666091919, -0.2398836314678192, -0.10910467803478241, -0.7974446415901184, 0.5407590270042419, 2.378960132598877, 1.2896369695663452, -0.5156596302986145, -1.493708848953247, 0.5684201717376709, -0.40867993235588074, -0.30322620272636414, -1.2022324800491333, -1.9227136373519897, -0.3816221356391907, 0.16811910271644592, 0.20796872675418854, 0.6634092926979065, -0.6038694977760315, 1.0860378742218018, 0.9704855680465698, -0.7040146589279175, 0.2869943678379059, 1.1129708290100098, 0.6796562671661377, -0.04328425973653793, -0.6300719976425171, -0.6313930749893188, -0.8472609519958496, -0.20194387435913086, 0.2389465570449829, 0.9416621327400208, 0.6829115748405457, -1.5635795593261719, -0.25587204098701477, -0.12357993423938751, -0.8520985245704651, 1.0881773233413696, 0.4745408594608307, 1.3671268224716187, 1.743126392364502, 0.6017523407936096, -1.542762279510498, -0.4394288659095764, -0.6776553988456726, -0.9586538076400757, 0.3842915892601013, 1.2446125745773315, 0.8807055950164795, -0.9848883748054504, -1.3649977445602417, 1.4700870513916016, -0.5817782878875732, 0.44979608058929443, 0.6722607016563416, -1.2885578870773315, -0.8538572192192078, -0.4968020021915436, -0.09451235085725784, 1.5111082792282104, -0.757744312286377, -0.17446008324623108, 0.2750459611415863, -0.7270087003707886, 0.2028048187494278, 1.1634060144424438, -1.324052095413208, -0.7997023463249207, -0.5110187530517578, 0.7862169742584229, 0.13486157357692719, -0.6039005517959595, 1.5824552774429321, -0.796089768409729, 0.3791237473487854, 0.37608587741851807, 0.5973010659217834, 1.1444754600524902, -0.4200997054576874, -0.3353082239627838, -0.36876264214515686, -0.11947352439165115, -1.766525387763977, 0.8464137315750122, -1.0581374168395996, 1.2900757789611816, 1.2289224863052368, 0.40707534551620483, 0.20511892437934875, 0.012146368622779846, -0.5621189475059509, -0.380748450756073, -0.5614645481109619, 0.14501038193702698, -1.3414679765701294, -0.5154504776000977, -0.5283284187316895, -0.8057083487510681, -0.025226330384612083, -0.27982404828071594, -0.7693358659744263, 0.7548913955688477, 0.8043102025985718, 0.8778738975524902, -1.5420949459075928, -1.6868197917938232, -1.440804123878479, 0.9768102765083313, -0.996105432510376, -0.664021909236908, -0.06550175696611404, -1.0091811418533325, -0.1064128503203392, 1.3174686431884766, -0.17304390668869019, 0.9261960983276367, 1.5333449840545654], [1.7902677059173584, -0.8600616455078125, 1.072457194328308, -0.37420397996902466, -0.46289485692977905, -0.10521931946277618, -0.7948971390724182, 0.6381773352622986, -0.04903338849544525, 1.398116946220398, -0.5230907201766968, 0.05799699202179909, -0.18787150084972382, -1.4400451183319092, -0.4593656659126282, -0.18170157074928284, -0.7472847104072571, 2.6815273761749268, -1.3958606719970703, -0.9150403738021851, -0.8462086915969849, -1.2451791763305664, 0.8355597853660583, 1.532516598701477, -0.3438907563686371, 0.67596435546875, -0.6734820604324341, 0.20107941329479218, -0.5299105048179626, -2.5801215171813965, -0.6751617789268494, 0.8098520040512085, -0.18599195778369904, -1.2859299182891846, 1.7317826747894287, 0.010323558934032917, 1.0876299142837524, -0.8233328461647034, 1.1645312309265137, -0.8245142698287964, 1.2385599613189697, -0.06410884857177734, 0.08284416794776917, -0.006217596121132374, 0.46188557147979736, 0.5314555764198303, -0.08668649941682816, 0.04495031386613846, 1.5360815525054932, -0.2832295894622803, -0.5467908978462219, -0.6882690191268921, 0.7377444505691528, 0.44819799065589905, 1.2207016944885254, -0.548121988773346, 0.2290629893541336, 0.40704768896102905, -0.14370769262313843, -1.5342135429382324, 0.9565713405609131, -0.6254227757453918, 0.5923303961753845, -1.468493103981018, -0.4735952615737915, -1.1387399435043335, 1.3672689199447632, 0.9404526948928833, 2.4640448093414307, -0.7303180694580078, 0.6087000370025635, 2.693321466445923, 0.8161460757255554, -0.9784628748893738, -0.07864212989807129, -0.5293123126029968, 0.8902027010917664, 1.5835881233215332, -0.5141177773475647, -0.6584720015525818, 1.4710465669631958, -1.3996425867080688, -0.001983328489586711, 0.5144360661506653, 0.09475010633468628, 1.1243928670883179, 1.212438702583313, 0.6553507447242737, 0.7988481521606445, -0.05212694779038429, 0.4773359000682831, -0.2559024691581726, -0.7648729681968689, -0.24962058663368225, -1.3546656370162964, 1.0489404201507568, 0.3978717625141144, -0.022785719484090805, -1.2095346450805664, -0.8199531435966492, 1.6828464269638062, -0.3567168712615967, 0.07167267799377441, -0.7750047445297241, -0.022060267627239227, 0.7950064539909363, -0.8726743459701538, -0.28800690174102783, 0.8179494738578796, -1.6491026878356934, -0.39034172892570496, 0.9005075693130493, 1.4919302463531494, 0.27709755301475525, 0.349538117647171, -0.2876955270767212, -2.1123008728027344, 0.4392566680908203, 1.0767663717269897, 3.502035140991211, 0.9762413501739502, 0.741608738899231, 0.7056394219398499, 0.07593270391225815, 0.19369985163211823, 0.8173706531524658, 0.17530982196331024, 0.5848337411880493], [-0.09202349185943604, -0.04766370728611946, 1.41873300075531, -0.9084949493408203, 1.114486575126648, 0.059019818902015686, -0.015250897035002708, 1.172580361366272, 1.6078159809112549, 0.40636616945266724, -0.6257662177085876, 1.805827260017395, -0.25550758838653564, 0.6449736952781677, 1.24158775806427, 0.838308572769165, 0.18936829268932343, 0.1419493854045868, -1.451363205909729, 0.15381494164466858, 0.11538632959127426, 0.719581127166748, -0.6049603223800659, 1.93975031375885, 0.9235051870346069, -0.5710073709487915, -0.18719355762004852, -0.9272704720497131, 1.175049066543579, 1.2197343111038208, -0.9095941185951233, 0.7051082253456116, -2.2535834312438965, -0.02475312165915966, 2.4877586364746094, -1.1495673656463623, 0.5260288715362549, 0.780444324016571, 0.31628695130348206, 0.15015757083892822, 1.589648962020874, 0.5574895739555359, -0.5951926708221436, -0.34904760122299194, -0.1643742471933365, -1.5051864385604858, 1.092111587524414, -1.9900325536727905, -0.5695357918739319, 1.0331789255142212, 0.9372453689575195, 0.248403862118721, 0.6545270085334778, 0.19730816781520844, 1.233594536781311, 1.0504133701324463, 1.4123724699020386, 2.150500774383545, -1.0702685117721558, 1.5252269506454468, 0.44558003544807434, -0.9249035716056824, -0.11511434614658356, 0.22946801781654358, 1.037536859512329, -1.6437597274780273, -1.1008286476135254, 1.008072018623352, 1.4097880125045776, 0.701411247253418, 0.8199907541275024, -0.8850074410438538, -1.7009224891662598, 1.1728769540786743, -2.4082071781158447, -1.3277807235717773, 0.9160820245742798, -0.24895833432674408, -1.5925440788269043, -0.931493878364563, -0.39602935314178467, 0.5186716914176941, 0.18028581142425537, 1.104949712753296, 1.1603002548217773, -0.43704530596733093, -0.8183920979499817, 0.3135431408882141, 0.1061503067612648, 0.01364124845713377, -0.3208558261394501, 0.10615633428096771, 0.09871798753738403, 0.011941621080040932, -0.5306981801986694, 0.558711051940918, -1.2314194440841675, -2.101494789123535, 0.2504585385322571, 0.07080474495887756, -1.3791006803512573, -0.875196635723114, -0.42888495326042175, -0.10551141947507858, -1.098841905593872, -0.08422104269266129, 1.4077684879302979, -0.9680721759796143, -0.4691725969314575, 0.27891889214515686, 1.3462841510772705, 0.7296552062034607, 1.4476090669631958, 0.3570573627948761, 1.1196786165237427, 1.349833369255066, 0.16512560844421387, -0.5860408544540405, -1.8534932136535645, -0.3239521086215973, -0.6949576139450073, -1.797341227531433, -1.6113975048065186, -0.5355669260025024, 0.13291075825691223, 0.026064183562994003, 1.4356873035430908, -0.33198466897010803], [1.5903942584991455, 1.4900168180465698, 1.0566189289093018, 0.5744363069534302, -0.205438032746315, -0.09259586036205292, 0.4867076277732849, 0.4194486439228058, 0.6419752240180969, 0.47485318779945374, 2.104374408721924, 0.14560788869857788, -1.6274324655532837, -1.1198084354400635, 1.0476478338241577, -0.8499729633331299, -0.5012799501419067, 0.6099533438682556, 0.5048015117645264, -0.5379908084869385, 0.858180582523346, 0.5248367190361023, -1.3110443353652954, 1.9917762279510498, 0.9273934364318848, -0.5067535638809204, 1.382103681564331, 0.7925697565078735, -0.36926987767219543, 0.425550639629364, -1.0372259616851807, 1.5796281099319458, -0.3028489649295807, -0.17565597593784332, 0.12940111756324768, -1.3626165390014648, 0.5196892023086548, 0.3500863015651703, 1.0966547727584839, -1.537950038909912, -0.2450786530971527, -0.9191237688064575, -0.5162444710731506, -0.9951047897338867, 2.0120456218719482, 0.7594829201698303, -0.7828044891357422, 0.36736616492271423, -0.40386223793029785, -0.1778487265110016, -0.4902411997318268, -0.03893347457051277, 0.5165543556213379, -1.2022937536239624, 0.9663018584251404, -2.0230867862701416, -2.1822423934936523, -1.9597574472427368, 0.5585449934005737, -1.3571364879608154, -0.7062016725540161, 1.3711967468261719, -0.8830928206443787, -1.1721855401992798, 0.14627543091773987, -1.561729073524475, 1.0955708026885986, 0.08722628653049469, -1.3422634601593018, 1.6183819770812988, -0.6066936254501343, -1.1859372854232788, -0.04388491064310074, 0.8393107652664185, 1.5681699514389038, 0.48020610213279724, 0.14421826601028442, 1.1935701370239258, -1.7843172550201416, 1.4030815362930298, -0.729292094707489, -1.5120820999145508, -0.1925508826971054, 1.3809137344360352, -0.5931978225708008, 0.44995245337486267, -1.8177887201309204, 0.5057402849197388, 1.7175430059432983, -0.457563191652298, -0.27155429124832153, 1.0072193145751953, -0.9670859575271606, 1.1477022171020508, 0.49191147089004517, -0.43291065096855164, 0.3215700685977936, 0.9497870206832886, 0.7453525066375732, -0.9910749197006226, -0.41000986099243164, -1.362868309020996, -0.010069933719933033, 0.40697383880615234, 0.15602025389671326, 0.25731074810028076, 0.9494842886924744, -1.0499452352523804, -1.2276842594146729, -1.4258627891540527, 1.1487210988998413, 1.0334326028823853, 0.7300562262535095, -1.461911916732788, 2.0718841552734375, 1.175328254699707, -0.28744828701019287, 0.31360146403312683, 0.442624568939209, -1.3938465118408203, 0.5966806411743164, 0.5411987900733948, -1.2550389766693115, -0.5473033785820007, 0.049141060560941696, -0.8241487741470337, -0.3789631426334381, 0.14060306549072266], [-0.7518917322158813, 0.9210746884346008, 0.19499856233596802, 0.8657514452934265, -0.12500108778476715, 0.11640320718288422, 0.9561702013015747, -0.4010991156101227, 0.34931066632270813, 0.6326578855514526, 0.7825428247451782, -0.14833679795265198, 0.098692886531353, -0.469153493642807, -0.7470259070396423, 1.0603801012039185, 0.1534217745065689, -0.2373417615890503, 1.6745842695236206, 1.2427493333816528, -0.7550076246261597, 0.271304190158844, -1.4725953340530396, 1.7798343896865845, 1.0936975479125977, 0.22345338761806488, -0.3814174234867096, 2.2268455028533936, 0.5025995969772339, -0.8648614883422852, 0.9430757761001587, -1.6347806453704834, -0.4030658006668091, -0.586696207523346, -0.6523153781890869, 0.3744755685329437, 0.18478171527385712, 0.6764146089553833, -0.11165209859609604, 1.2334176301956177, 0.5565888285636902, 0.2671789526939392, 2.497119665145874, -1.0652114152908325, 1.8119540214538574, -0.7467882633209229, 0.10739995539188385, 1.7898049354553223, -0.8793987035751343, -1.164115309715271, 0.35033246874809265, 0.19758537411689758, 1.633601188659668, 1.1251434087753296, 0.11062787473201752, -1.1312689781188965, -0.1668599396944046, -0.8391612768173218, 0.42228105664253235, -0.4511169195175171, -0.42542916536331177, 0.06154516711831093, -0.796680748462677, 0.7866137623786926, 0.3142858147621155, -0.48506322503089905, 0.11863772571086884, 0.9910622835159302, -0.8957634568214417, 0.11203634738922119, 1.3855564594268799, 2.528214931488037, 1.0030713081359863, -1.088869333267212, 0.18595802783966064, -0.7877805233001709, -1.5601195096969604, 0.1512330174446106, -0.4223840832710266, -1.3692138195037842, 0.48504799604415894, 1.4483810663223267, 1.0313360691070557, 0.409124493598938, -0.5901015400886536, -0.8939590454101562, -0.5540136098861694, 0.2129725068807602, -1.0445680618286133, 2.1156394481658936, -0.26935043931007385, 0.7539967894554138, -0.29227951169013977, 0.6117953062057495, 0.11565303802490234, -0.049794070422649384, -1.351365089416504, 0.5156105756759644, 0.9365181922912598, -0.185980424284935, 0.6111329793930054, 0.5951264500617981, -0.9287815690040588, 0.4197794795036316, -1.3066630363464355, -0.36932432651519775, 0.9099190831184387, -1.5855495929718018, -0.21304164826869965, 0.19901655614376068, 0.6566834449768066, -0.46709251403808594, 1.5014331340789795, -0.2574670612812042, 1.28882896900177, 1.832903504371643, 2.5334761142730713, 0.7330537438392639, -0.1219152957201004, -0.03358970209956169, 0.18002460896968842, 2.4547030925750732, -0.851096510887146, 1.3924925327301025, 0.17391085624694824, 0.04069085791707039, 0.48564451932907104, 1.3884828090667725], [-0.7185223698616028, -0.10975229740142822, 0.5003179907798767, -1.0512831211090088, 0.5842300057411194, -0.28051674365997314, 0.6746966242790222, 2.321455478668213, -0.22611123323440552, 1.918437123298645, 0.9904688596725464, 0.5452885031700134, 0.19296786189079285, 0.8051660060882568, -1.3307271003723145, -0.7537273168563843, -1.6232904195785522, 1.4981411695480347, -0.981278121471405, -0.9672251343727112, -0.23595553636550903, -0.07859242707490921, 0.8186237812042236, 0.8362865447998047, -1.2440721988677979, -1.8942828178405762, -0.5809401869773865, 2.2764179706573486, -2.161102056503296, -0.38257071375846863, -0.5764288306236267, -0.31968429684638977, 0.10971404612064362, -0.008604073897004128, -0.4292491376399994, 0.6744655966758728, -0.4143376350402832, -0.2756979763507843, 0.7178928256034851, -0.2169192135334015, -0.5515736937522888, -1.4889274835586548, 0.011065466329455376, -0.9427066445350647, -0.6966431140899658, -0.24247701466083527, -1.2009004354476929, 0.5251671075820923, -3.0100138187408447, 0.6720764636993408, -1.5983190536499023, -1.508880853652954, 0.47930213809013367, 1.7021517753601074, 0.8142974376678467, 0.18825992941856384, -0.8057754635810852, -3.0404698848724365, 0.18514414131641388, 1.6033226251602173, -0.7617986798286438, 0.22084875404834747, 0.998749315738678, 0.4309185743331909, -0.7291759252548218, 0.5602255463600159, -0.07292235642671585, 0.2561185657978058, -1.2104592323303223, 0.7022861838340759, -0.6585028171539307, -0.46855294704437256, 0.2621687352657318, -0.045018892735242844, -0.07800734788179398, 1.2383325099945068, 1.7002087831497192, 0.31165972352027893, 0.43681713938713074, -0.45770731568336487, 1.5923947095870972, 0.1939605325460434, 0.6735012531280518, -1.3989295959472656, -0.7420294284820557, -0.12171482294797897, 0.7245028018951416, -0.7058582305908203, -2.3637518882751465, 0.1300664097070694, 0.7527573704719543, 1.3489267826080322, -1.8244500160217285, -0.2565573453903198, 1.1013846397399902, 0.5864878296852112, -0.5790593028068542, 0.3008767366409302, -1.8994563817977905, 0.18347260355949402, -0.8244454860687256, 0.35936057567596436, 0.22630269825458527, -0.05964818224310875, 1.0166488885879517, -2.2438724040985107, 0.7660902142524719, 1.514302134513855, 1.5222336053848267, 0.7884762287139893, 1.614651083946228, -0.09152156114578247, 0.6575332880020142, 0.7852333784103394, -2.0757334232330322, 0.22549179196357727, -0.5164425373077393, -0.8598861694335938, -1.8141242265701294, 2.2162013053894043, 0.9586371183395386, -0.22241589426994324, 0.8479713201522827, 0.21619868278503418, -1.5958877801895142, -0.31858909130096436, -1.4911999702453613, -0.9879659414291382], [-0.22926698625087738, 0.26576879620552063, -0.9770460724830627, -0.8058613538742065, 1.5514991283416748, -0.2535346746444702, 0.30044326186180115, 1.2559125423431396, 1.4135688543319702, 1.1324036121368408, -1.2346612215042114, -0.8533157706260681, -0.9530947804450989, 1.582907795906067, 1.2410914897918701, 1.3343901634216309, 1.338321328163147, -0.9566812515258789, 1.2548918724060059, 0.058459796011447906, 1.2122770547866821, -2.447451591491699, -0.724055290222168, -0.2798381447792053, -0.07567849010229111, -0.6031134128570557, -0.3741656243801117, 0.28539538383483887, -0.6501697301864624, -2.392103910446167, -1.17593514919281, -1.2672853469848633, 0.08693437278270721, 1.2598713636398315, 0.24791860580444336, 0.7532559633255005, 0.48743072152137756, -0.4819101095199585, -0.45714959502220154, 0.8401210308074951, 0.07718914747238159, 1.0661646127700806, 1.0728962421417236, 0.5681394338607788, -0.7777152061462402, 0.8081772923469543, -0.28199148178100586, 1.2770966291427612, -1.460924744606018, 1.5739929676055908, 0.34626272320747375, 1.0687007904052734, 0.27875611186027527, -0.31823334097862244, 0.4850054979324341, -0.6129570603370667, -0.9539574980735779, -0.9962078928947449, -2.1369917392730713, -0.8318299651145935, 1.893736720085144, -2.4774892330169678, -1.1987289190292358, 0.6292533278465271, 0.6290555000305176, -0.5923904776573181, -0.2981867790222168, 1.5141764879226685, -1.2284824848175049, -0.38823801279067993, 0.7634198069572449, 0.9889682531356812, 0.32580843567848206, 0.03170550987124443, -1.4545934200286865, -0.11359179764986038, -1.3062089681625366, 0.37705880403518677, -1.3388080596923828, 0.8278568387031555, 1.503798007965088, 1.6432589292526245, 1.1020103693008423, -0.05006997659802437, 0.26306334137916565, -0.524350106716156, -0.8913697004318237, 0.5973702073097229, 0.12496118247509003, 0.6845517754554749, 0.3509814441204071, 1.0291491746902466, -0.506675660610199, -0.08772461861371994, -1.0670878887176514, -0.6326815485954285, -0.6521430015563965, -0.8303872346878052, 0.7651357650756836, -0.8132141828536987, -0.7946163415908813, -1.5918351411819458, 0.4132835566997528, -2.5190200805664062, -0.4281410872936249, 0.9230989217758179, 0.6707973480224609, 0.7736412286758423, 0.6079687476158142, 1.178780436515808, -0.9495458602905273, -0.09101822227239609, -0.1687919646501541, -0.4415953755378723, -0.6056520938873291, 0.028814053162932396, 0.26236817240715027, 0.9879337549209595, -0.7153169512748718, -0.3673912584781647, -2.231806993484497, 1.8512595891952515, -0.3866112232208252, -0.11159133166074753, -1.0271002054214478, -0.010666904971003532, -2.096496343612671, -0.5191658735275269], [-0.2953221797943115, -1.0696617364883423, -1.6620001792907715, 0.8303449153900146, -0.8993768095970154, 0.6082672476768494, 0.2569587528705597, -2.292165517807007, -2.3788340091705322, -0.34327980875968933, -0.928226113319397, -0.030967650935053825, -0.8433026671409607, 0.9026267528533936, 1.5717864036560059, -1.0850038528442383, -0.4196927845478058, 2.373236894607544, 0.6186770796775818, -1.6381508111953735, 0.7366304993629456, -0.8013069033622742, -1.3043498992919922, -0.6264142394065857, 0.8735829591751099, 0.6763916611671448, -0.6976483464241028, -0.13718947768211365, 0.8328666687011719, -0.40633347630500793, -0.4992671310901642, 0.3423343002796173, 0.785515546798706, 0.027916885912418365, -1.3008079528808594, 0.6406434178352356, -0.5595689415931702, -0.848585844039917, -0.3262525796890259, -0.9910727739334106, -1.6005501747131348, -0.4274330735206604, 0.7722243070602417, 2.0637106895446777, -0.40951037406921387, 0.14862164855003357, 0.8706102967262268, -0.5213263034820557, 0.025123709812760353, 1.0173906087875366, -0.8336016535758972, -1.6735167503356934, -1.6201974153518677, 1.647637128829956, -0.11291006952524185, 0.0065787178464233875, -0.07867616415023804, 1.9948234558105469, 0.5193201303482056, -1.836928129196167, 1.0268975496292114, -1.1026662588119507, -1.9341049194335938, -1.3335455656051636, 0.23655354976654053, 0.4553055763244629, -0.6538986563682556, 0.04911283776164055, 1.169878602027893, 0.15773330628871918, -0.4074617326259613, -0.9083644151687622, -0.7088786959648132, -0.9072582721710205, 0.21972781419754028, -0.9709132313728333, -0.27260667085647583, -0.49151960015296936, -0.9406578540802002, 0.80628502368927, 0.47440066933631897, 0.31413987278938293, -1.113857388496399, 0.01771152764558792, -1.872890830039978, -1.1983054876327515, -1.2945175170898438, -0.6676816344261169, 0.4430968165397644, 0.1822706162929535, -0.19962657988071442, -1.1212882995605469, -0.13193733990192413, -1.9584791660308838, 1.5104202032089233, -0.2526598870754242, 0.25989478826522827, -0.467275470495224, 0.819705069065094, 0.7186668515205383, 0.3803698718547821, -0.024744706228375435, -3.2535409927368164, 0.497211217880249, -2.566896438598633, 1.654254674911499, -0.8787355422973633, -0.708268404006958, -1.0977269411087036, -0.6216430068016052, -0.5852319002151489, 0.5553910136222839, -0.47746339440345764, -0.3318956792354584, -1.2245539426803589, 0.02193758450448513, -0.12618613243103027, 0.8833476305007935, 0.4772108495235443, -0.2195216715335846, -0.05663658306002617, 0.3583395481109619, 2.0447068214416504, -2.4648923873901367, -0.517694890499115, -1.0328980684280396, -0.7295047044754028, -0.044844359159469604], [0.16378791630268097, -0.4584110975265503, 0.16754025220870972, -0.44355326890945435, -0.9233525991439819, -1.2761282920837402, 1.2567304372787476, 0.213611900806427, -0.6895360350608826, -1.1253657341003418, -2.2933170795440674, 0.06534866988658905, 1.1466916799545288, 0.1887618750333786, -2.304779052734375, -0.695270836353302, -0.011158107779920101, 0.16161029040813446, 0.7768205404281616, -0.5404585003852844, 1.1590418815612793, -0.5927801132202148, -0.9094085693359375, -1.1196104288101196, 0.25957930088043213, 0.10720714926719666, -0.605186402797699, 0.4501955211162567, -0.5428663492202759, -0.10511258244514465, -0.5458469986915588, 0.7530606389045715, -0.5070696473121643, -0.6197211742401123, 0.9286562204360962, 0.6022870540618896, -1.9572595357894897, -0.45796456933021545, 0.46299436688423157, -0.1533975452184677, 0.4150468409061432, 0.02607031911611557, 0.48294514417648315, 1.4121016263961792, -1.0463210344314575, 0.2636640667915344, -0.4178420603275299, 1.8874576091766357, -0.6053076386451721, 0.17979265749454498, 1.3910290002822876, -1.187385082244873, 0.6092238426208496, -1.000664472579956, 0.5387405753135681, -0.48858344554901123, 0.37234795093536377, -0.49949297308921814, -0.8962505459785461, 0.08865074068307877, -1.0249325037002563, 1.0615777969360352, -0.3221290707588196, -0.8725204467773438, 0.4009345769882202, -0.37729158997535706, -0.13091714680194855, -1.3513668775558472, -0.2911394536495209, -0.3336041271686554, -0.9428212642669678, 0.40424835681915283, 0.12357375025749207, -0.9580291509628296, 0.5204582810401917, 0.19663335382938385, 0.30360743403434753, -0.3766286075115204, -0.045833759009838104, 0.2292884886264801, 0.12144719064235687, 0.021787187084555626, -0.730958878993988, 0.30475014448165894, -0.9186515808105469, 1.3215008974075317, -0.6077376008033752, 1.2585039138793945, 0.8864995241165161, 0.2569499611854553, -0.08022081851959229, -0.34485745429992676, -1.0665173530578613, -0.7294355630874634, -0.20160824060440063, -2.0551705360412598, 1.2423244714736938, 1.8828336000442505, -0.11663582921028137, 0.3442809283733368, 0.49424684047698975, -0.9955662488937378, -1.0289268493652344, -1.2224299907684326, -1.9398040771484375, 0.4572139084339142, -0.931504487991333, 1.3424383401870728, 1.826429843902588, -0.7319933176040649, 0.24476562440395355, -0.6402842402458191, 0.9986189603805542, -0.42373859882354736, -0.840071439743042, 1.4046707153320312, 0.9253146052360535, 0.4384188652038574, 0.1967240273952484, -0.43917810916900635, 0.21962983906269073, 2.6780283451080322, -0.2893528640270233, 1.1525707244873047, 0.6881822347640991, 0.27143797278404236, 0.3291761875152588, -0.32642972469329834], [-0.34505102038383484, -0.003037462942302227, 0.4883746802806854, -0.29482555389404297, 0.2829018831253052, 1.1283330917358398, -0.8039456605911255, -0.4763420820236206, -0.39625540375709534, -0.28293150663375854, 0.1839992254972458, 1.8108890056610107, -0.0033615026623010635, 1.1481378078460693, 0.6540058255195618, 0.43931323289871216, 0.824210524559021, -1.6664149761199951, 1.0536845922470093, -1.7575026750564575, 1.002339243888855, -2.0430212020874023, 2.0096421241760254, 1.0243821144104004, 0.5399798154830933, -0.3958326280117035, 2.1131837368011475, 0.4902368485927582, -1.4510527849197388, -0.8700461983680725, -0.40200749039649963, -0.8579777479171753, -1.8162506818771362, 0.3406946659088135, 0.09124022722244263, -0.7706325650215149, -0.030118733644485474, 1.881489872932434, -0.5338639616966248, -0.5510729551315308, 0.308960497379303, -1.0604729652404785, -0.4152681231498718, -1.3505783081054688, -0.7769007086753845, 0.6345183849334717, -0.950559675693512, 1.003920316696167, -0.14898879826068878, 1.33234703540802, -0.5981993675231934, -0.4060220718383789, 0.55247563123703, 0.00956935714930296, 1.3992925882339478, 0.19697077572345734, 0.10228358954191208, 1.8792436122894287, -0.48035356402397156, 1.9100158214569092, -0.66762375831604, 0.9942111968994141, -1.3983060121536255, -0.7083866000175476, -0.6139249801635742, -0.8335482478141785, -1.5835940837860107, 0.9893139600753784, -1.9328312873840332, -0.6732762455940247, -0.9071871042251587, -1.4480559825897217, 0.5475286841392517, 0.5089933276176453, 0.25379931926727295, 1.7462413311004639, -0.06148763373494148, 1.7557286024093628, -1.1666587591171265, -0.5782493948936462, -0.8182551860809326, 2.8453168869018555, -0.0262156929820776, 0.20684000849723816, 1.3438259363174438, 0.4479886293411255, 0.627325177192688, -0.04076271876692772, 0.22800925374031067, -1.7477229833602905, 0.6662495136260986, -1.5144054889678955, 0.42478808760643005, -0.5468178987503052, -1.1803579330444336, 0.439435213804245, -0.6046075224876404, 0.9254411458969116, -1.6204252243041992, -0.2437065690755844, 0.12153754383325577, 0.46653762459754944, -0.44668474793434143, -0.14539207518100739, -0.19088204205036163, 0.4157091975212097, 0.10386735945940018, -0.850509762763977, -1.1152215003967285, 0.4579741358757019, 0.6481136679649353, 1.3113114833831787, -0.5059801936149597, -0.09349023550748825, 0.17236506938934326, -0.7130045294761658, -0.4656654894351959, -0.18496324121952057, 1.6302721500396729, 0.1957959532737732, 0.8352944850921631, 0.7150586843490601, -0.8761134743690491, -0.985167920589447, 1.728888988494873, -0.10546212643384933, -1.0524660348892212, -0.9482658505439758], [1.3474721908569336, -0.8759922385215759, -0.7798674702644348, -1.0147653818130493, -0.6373092532157898, 1.7868459224700928, 1.1993319988250732, -0.36731916666030884, -1.195749282836914, -0.7484087944030762, 1.3968236446380615, 0.07657185196876526, 1.1571147441864014, -0.20060735940933228, 0.16612184047698975, -0.04969331994652748, 2.6935877799987793, -0.6765444874763489, -0.2901894748210907, -0.3955981135368347, 1.943154215812683, 1.4150797128677368, -0.32786446809768677, -0.9924272298812866, -0.5603196620941162, 0.6028696298599243, 0.5525436401367188, 1.1147310733795166, 1.3268190622329712, -0.2430618554353714, -1.3113772869110107, -0.20794253051280975, 0.6937062740325928, -1.2670931816101074, -0.10688843578100204, 2.4285967350006104, -0.007833759300410748, -0.7988967299461365, 1.0670654773712158, -1.1054116487503052, -2.178816080093384, -0.050917696207761765, 2.0544865131378174, -0.47317588329315186, 1.6714026927947998, 1.5310447216033936, 1.3874603509902954, -1.1399619579315186, 0.046359967440366745, -0.9368552565574646, -0.4329649806022644, 0.8719385266304016, 0.672216534614563, 1.9977551698684692, -1.062576413154602, 1.406820297241211, 0.8107001781463623, -0.950905978679657, -1.0707602500915527, 0.7604035139083862, 0.570115327835083, 0.3973119556903839, 0.6034833788871765, -0.41222238540649414, -1.4661411046981812, 0.9933447241783142, 1.4650448560714722, 1.535648226737976, 0.1180860847234726, -0.4075597822666168, 0.6985813975334167, 0.5692253112792969, -0.0713668093085289, -0.1629633754491806, 0.5834647417068481, -1.0149675607681274, 0.8066065311431885, 0.6084391474723816, -0.9297232627868652, 0.5619082450866699, -0.3491954505443573, -0.41262269020080566, 1.2298718690872192, 0.3314247727394104, -1.3635077476501465, 0.7467169761657715, 0.5247035026550293, -0.4139685332775116, 1.716849446296692, 0.8703598380088806, 6.284617120400071e-05, -1.2186782360076904, -0.6240766048431396, 0.5336838364601135, -3.3652706146240234, 0.3919963240623474, 0.5580869317054749, 1.249700903892517, 0.13532160222530365, 0.026446567848324776, -0.8858373761177063, 1.6967703104019165, 0.08616732805967331, 0.7040128111839294, -0.8285614252090454, 1.118277907371521, 0.3988840878009796, 0.544641375541687, 0.9754824638366699, 0.27724024653434753, 0.37250879406929016, 0.07574715465307236, 0.6382827162742615, 0.14608077704906464, -0.9482481479644775, 0.15281552076339722, 0.4041568636894226, -1.1466686725616455, -1.7949713468551636, -2.5566906929016113, 0.006566207390278578, -0.4941144287586212, -0.44096022844314575, 1.3417762517929077, 2.235830545425415, -1.9675780534744263, 2.3930656909942627, 0.04707964509725571]], [[-0.5481021404266357, 1.2931324243545532, 1.7917404174804688, -0.028777332976460457, 0.7448945641517639, 0.6895766258239746, 0.0004752935783471912, -0.344056099653244, -1.9829233884811401, -0.7669792771339417, 1.1504671573638916, -0.828068733215332, 0.6100836396217346, -0.9952414631843567, 0.0983884409070015, -1.050389051437378, 1.3202764987945557, -0.12769420444965363, 0.013926916755735874, 1.6366245746612549, 0.8233870267868042, -0.7907456755638123, 0.1521361917257309, -0.38366085290908813, -0.07443694770336151, 0.07372693717479706, -0.35543322563171387, -1.1636687517166138, -1.4760874509811401, 0.032509658485651016, -0.3112775385379791, -0.6367717385292053, -0.27860310673713684, -0.2278386801481247, -0.6607841849327087, 2.3365318775177, -1.6087933778762817, 0.23152437806129456, -0.3091737926006317, -0.1620398908853531, 0.688499391078949, -0.534075915813446, -0.9944450855255127, -0.5878122448921204, -0.9825341701507568, -1.8161225318908691, -0.9623957276344299, 0.8421622514724731, 1.2091500759124756, -2.2370948791503906, 0.25019168853759766, -0.3109731376171112, -0.2797167897224426, 0.300191730260849, 0.37744462490081787, 0.48201635479927063, 0.29485276341438293, -1.3280352354049683, -0.8272234201431274, 0.21691344678401947, 0.912106990814209, 0.10629278421401978, -0.06582733988761902, 2.0584583282470703, 0.18557694554328918, -0.7554872632026672, 0.6281910538673401, -0.16262757778167725, 1.5588771104812622, -0.5117061138153076, -0.4425382614135742, -0.08309432119131088, 0.5007951259613037, 0.21309539675712585, 1.639623999595642, 0.34927770495414734, 0.4571411609649658, -1.0161489248275757, -1.5182390213012695, -2.959958791732788, 0.2979356348514557, 1.08565092086792, 0.4863487780094147, -1.3375447988510132, 0.5582208037376404, 1.1429142951965332, -0.21040062606334686, 0.18683291971683502, -0.587416410446167, -1.6752859354019165, 0.9043737053871155, 1.2008877992630005, -1.0958423614501953, -0.6569505333900452, -0.5256557464599609, 0.5031319260597229, 0.24546191096305847, -1.8848174810409546, -1.2904982566833496, 0.4216367304325104, 1.8127835988998413, 0.9913926124572754, 1.132828712463379, 1.2655521631240845, 0.22912469506263733, 0.09234011173248291, -0.11181281507015228, 1.6159981489181519, -0.4185773432254791, -0.11815045773983002, 0.03761201724410057, 1.8484234809875488, -0.17853949964046478, 0.22571314871311188, -0.35914361476898193, -0.007079745642840862, 2.170827627182007, 0.26397794485092163, -1.0478589534759521, 0.4626457095146179, -1.4671802520751953, 0.19946737587451935, 1.2205365896224976, 0.4556657671928406, -0.5382013320922852, 0.5174804329872131, -0.6391327381134033, 0.04225875064730644], [-0.36876776814460754, -1.0241657495498657, -1.5727864503860474, 0.6946365237236023, -0.32410508394241333, -1.9280568361282349, -2.059499502182007, 0.4755781888961792, -1.5257760286331177, 1.8858896493911743, 0.18892760574817657, -1.5959864854812622, 0.23119443655014038, -2.6338930130004883, 0.23325559496879578, -1.0596786737442017, -0.17938633263111115, -0.6596671342849731, -0.45754337310791016, -0.7668899893760681, -0.5145376324653625, -0.5952503681182861, 1.6827367544174194, -0.8892515301704407, 1.849172592163086, 0.9225156903266907, 0.48199009895324707, 2.3135392665863037, 1.0569313764572144, 1.71425461769104, 0.4750460982322693, 2.339411735534668, 0.26789847016334534, -0.3163278102874756, 0.3925492763519287, 0.641217827796936, -0.052463896572589874, -0.06952560693025589, -1.0932953357696533, -0.5050653219223022, 0.3397448658943176, -0.11291101574897766, 0.7924461364746094, -0.27161839604377747, -0.638130784034729, 0.6204631328582764, 0.05096830427646637, -0.23311497271060944, -1.3112199306488037, -0.10143060982227325, 1.1356120109558105, -1.1438267230987549, 1.4891290664672852, -1.2739946842193604, -0.6140096187591553, 1.3222883939743042, 0.6075791716575623, 0.33536282181739807, -0.19939598441123962, 1.4916660785675049, -2.0841903686523438, -0.3151245713233948, -2.032475233078003, -0.7571187615394592, -0.2446264773607254, -1.1065667867660522, -0.6403936743736267, -0.6767556071281433, 1.9730027914047241, -1.5591446161270142, -0.3097764551639557, 1.3209718465805054, 0.1371380239725113, -0.14295968413352966, -1.0672742128372192, 1.8411897420883179, -2.0187864303588867, -0.6034649014472961, -0.43985655903816223, -2.039832353591919, -0.45357730984687805, -0.7093672156333923, 1.3770902156829834, 1.6289902925491333, -0.743317186832428, -0.0664781779050827, 1.5464996099472046, 0.3507528305053711, -0.6335494518280029, -0.2752893269062042, 1.1771502494812012, -0.8741303086280823, 0.9359427690505981, 0.14432473480701447, 1.0856388807296753, -0.466016948223114, 0.24758632481098175, 0.7476184368133545, 2.0684711933135986, 0.27745845913887024, -1.770835518836975, -0.7834362983703613, 0.6192105412483215, 1.0012989044189453, 0.149455264210701, -0.40794047713279724, 0.4241218864917755, -1.4078196287155151, -1.6771275997161865, -0.4096115529537201, -0.1770600527524948, -0.2848628759384155, 0.1943933516740799, 0.3148203194141388, 0.6846407055854797, 0.7111793756484985, -0.0028199872467666864, -0.25678807497024536, 0.5755349397659302, 1.9291343688964844, -1.3377466201782227, -0.40516793727874756, -0.18397359549999237, -1.0066413879394531, 0.5115907788276672, 1.2050037384033203, -0.19138723611831665, -0.08895868062973022], [0.7265992164611816, 0.7359225153923035, 0.732424259185791, 0.1308438777923584, -0.15392324328422546, 0.9023619294166565, -0.5322606563568115, 0.1711188107728958, 1.1564635038375854, -0.8807904720306396, -1.7057517766952515, 1.5690568685531616, -0.5337391495704651, -0.3194810450077057, -0.6493560075759888, 0.3067457973957062, -0.7848221659660339, -0.10073472559452057, 0.021221928298473358, -0.04063030332326889, -1.5805068016052246, 1.1278401613235474, 1.2690131664276123, -0.7318407297134399, 0.10927167534828186, -2.091998338699341, -0.6142001748085022, 0.5649980902671814, -1.3168792724609375, -1.6195144653320312, 0.6832588911056519, -2.012134313583374, 0.6078901290893555, -2.49422025680542, -1.365957260131836, 1.3168673515319824, -1.4785901308059692, 0.38306620717048645, -0.47404560446739197, 0.13393108546733856, -0.9119183421134949, -0.3077881336212158, 0.44498559832572937, 0.3723990023136139, 0.23156605660915375, -1.1736027002334595, 1.1157691478729248, -1.05182945728302, -0.17892929911613464, 0.06644808501005173, -0.08216403424739838, 1.3216769695281982, 0.1968885064125061, 0.724151074886322, 0.6480560302734375, 1.0284571647644043, 0.9745965600013733, 0.9687417149543762, 0.30550581216812134, 0.016404645517468452, 0.5136182308197021, 0.36771902441978455, 1.3886163234710693, 1.5602883100509644, 0.2990257143974304, 2.393434762954712, 0.4974721670150757, 0.42474913597106934, 0.5866863131523132, -0.6587846875190735, -0.0428227074444294, -0.9335709810256958, -0.5206257700920105, 1.3491638898849487, -0.5163111686706543, 0.3098006844520569, -0.2788456380367279, 0.1325571984052658, 0.6719131469726562, -0.418702095746994, -0.5425645112991333, 0.3574795722961426, 0.01617928221821785, -0.6923099160194397, -0.45880264043807983, 0.9591224193572998, 0.3252275288105011, -0.9435012936592102, -0.49563300609588623, 0.5192071199417114, -0.4146849811077118, 0.27163106203079224, -0.20322105288505554, 0.10361848771572113, 2.2306268215179443, -2.5444529056549072, 1.379323959350586, -0.4258768856525421, -0.59616619348526, -0.3090880215167999, -0.9604876637458801, 0.5932836532592773, 0.8659897446632385, -0.5877490043640137, -0.5189440250396729, -0.4481392502784729, -0.9283779263496399, -0.8526532053947449, 0.08694762736558914, 0.14507821202278137, -0.19077004492282867, 0.9535399675369263, 0.7180631160736084, -0.7474574446678162, 0.21054936945438385, -0.6103120446205139, -1.1072609424591064, 0.26627394556999207, 0.2262817770242691, -2.1765177249908447, -0.4672377109527588, -0.7569761872291565, 0.05565500259399414, -1.8480424880981445, 1.1718873977661133, 0.5307679176330566, 1.0323681831359863, 0.2396855652332306], [-0.9964331388473511, -1.2686306238174438, 2.0322799682617188, 0.5060275793075562, -0.4719116985797882, -0.8670381903648376, -2.4288859367370605, 0.21192337572574615, -2.498668670654297, 0.49061161279678345, -0.9100633859634399, -0.7670994997024536, 0.9722578525543213, 1.1012067794799805, 0.74711012840271, 1.298959493637085, -1.1211845874786377, 0.32352206110954285, -1.1404155492782593, -0.7005730867385864, -0.44146791100502014, -0.2810838520526886, 1.4729063510894775, -0.6525461673736572, -1.6958528757095337, -0.7533541917800903, -0.5225030183792114, -0.4689854383468628, -2.115187406539917, -1.795161485671997, 1.12446928024292, -0.12260263413190842, -1.1854599714279175, 2.9413623809814453, -0.8698139190673828, 0.5726019740104675, 0.0057142446748912334, 1.5403847694396973, 1.1022696495056152, 0.1870884895324707, 0.46482375264167786, 1.3473623991012573, -0.7133379578590393, -1.2159650325775146, 2.1895651817321777, 0.7493035197257996, 2.585712194442749, 0.8955857157707214, -0.6868404150009155, -0.6845917701721191, -0.22702999413013458, 0.015105138532817364, 0.3067479431629181, -0.579558789730072, 0.87709641456604, 0.17944969236850739, 0.464896023273468, 1.7308026552200317, -1.4528381824493408, -0.02947336807847023, -0.5414065718650818, -0.8740708231925964, -0.5151918530464172, -1.048326015472412, 1.5191367864608765, -1.633719563484192, 0.2922186255455017, 0.18069906532764435, -1.418458104133606, -0.28739604353904724, -0.8700345754623413, -0.3027810752391815, 1.4321469068527222, 1.6401934623718262, 0.6860707998275757, 0.08525282889604568, -0.6390341520309448, -0.9468328356742859, 0.09268992394208908, -0.011520516127347946, 0.40890219807624817, 0.542729377746582, 1.2369402647018433, 0.704623281955719, -0.2963671088218689, 1.9304828643798828, -0.3392910659313202, -1.5430227518081665, -0.4111214578151703, 0.5284344553947449, -1.1446596384048462, 0.46358054876327515, 0.8382394909858704, 0.14660999178886414, 0.700352668762207, -0.6081067323684692, 0.1390477567911148, -0.45887327194213867, 0.12092141807079315, 0.09919367730617523, 1.7117811441421509, 1.3084772825241089, 0.25564509630203247, -0.5899479389190674, 0.5350595116615295, 0.6280379295349121, 0.6278588175773621, -1.9078519344329834, 1.4798461198806763, -0.2746722996234894, -0.07968684285879135, -0.37389716506004333, 0.05283234268426895, -2.4437193870544434, -1.4491983652114868, 1.959932565689087, -1.1545974016189575, 0.812200129032135, 1.7824697494506836, -0.08950787782669067, 0.15105843544006348, 0.4106367230415344, 0.2208739072084427, -1.7156078815460205, -1.25559401512146, -0.02886594645678997, 0.9947463870048523, 0.8001166582107544], [-0.7975097298622131, -1.521856427192688, -2.040396213531494, -0.6837195158004761, 0.10624100267887115, 0.6970321536064148, -0.49432238936424255, 0.761494517326355, -0.807778000831604, 0.20004601776599884, 0.0678304061293602, 1.2400683164596558, 1.5404495000839233, 0.7147549986839294, -2.868605613708496, -2.114206552505493, -1.8051003217697144, 1.0114669799804688, -0.09473962336778641, 3.3233213424682617, -0.15260086953639984, 1.3193824291229248, -0.7477172613143921, 0.14391696453094482, 0.6836764812469482, 2.126828670501709, 1.9244639873504639, -3.1071460247039795, 0.20229299366474152, 1.1944642066955566, -0.503358006477356, 0.2793639004230499, -0.4535420536994934, 1.9479546546936035, -1.4290130138397217, -0.4315926730632782, 1.3101197481155396, -0.7817975878715515, 0.8332035541534424, 0.6621158123016357, 0.8791985511779785, -0.20286117494106293, -0.3026263415813446, 0.9300835132598877, -0.7968422770500183, -1.2120325565338135, -0.7310217618942261, 0.5157903432846069, -1.1081211566925049, -0.9427894353866577, 0.6567748188972473, -0.3564400374889374, -1.0878764390945435, 0.632727324962616, 0.31196755170822144, -1.4074510335922241, -1.5899420976638794, -0.20966243743896484, -0.3707263469696045, -0.907480776309967, -1.3733569383621216, -0.6193253397941589, 0.5127415060997009, -0.8814355134963989, 1.3153952360153198, -0.11502107977867126, -0.3765951991081238, -1.929105281829834, 0.27877509593963623, -0.8617697358131409, -1.4508817195892334, 0.39621275663375854, 1.2011805772781372, -0.2571181356906891, -0.6115949153900146, -0.0686948299407959, -1.1712595224380493, -0.4629344940185547, -0.9876110553741455, -0.9174129962921143, -0.181392103433609, -0.46832117438316345, 1.4775280952453613, 1.9229600429534912, -0.09374719113111496, 0.0661044791340828, -0.4347619414329529, 1.113511323928833, -0.4897801876068115, 1.2095048427581787, -1.6822524070739746, 0.5524266362190247, -0.8825421333312988, 1.081701397895813, -0.926170825958252, -0.8578268885612488, 0.2771642506122589, -2.0143868923187256, 1.7802690267562866, 2.269749402999878, 0.21912717819213867, 0.3301081955432892, -0.4386764466762543, 0.0024830219335854053, 0.05169457942247391, 0.5786466598510742, 1.0797377824783325, -0.42070016264915466, -1.7439225912094116, -0.13411514461040497, -0.03882547840476036, 1.1347907781600952, -0.7178145051002502, -0.015782926231622696, -1.736441969871521, 1.7728246450424194, 0.632978081703186, 1.1536743640899658, 0.5705642700195312, -2.5204756259918213, 1.0993757247924805, -0.12915897369384766, -0.9130370020866394, 0.6313616037368774, 0.524010956287384, 0.7200388312339783, 3.11786150932312, 2.2083239555358887], [-0.050952207297086716, 0.4739473760128021, 0.11147011816501617, 0.9358076453208923, -2.172903060913086, 0.11993762105703354, -1.4990700483322144, 1.875697135925293, -0.5284321308135986, -0.13096393644809723, -0.39776843786239624, -3.6267693042755127, 0.006848156452178955, -0.570149838924408, 1.7095533609390259, 0.5094243884086609, 2.1854472160339355, -0.6077407598495483, -0.7646856307983398, 0.5514299869537354, 0.7542220950126648, 0.08320725709199905, 0.33726227283477783, -0.7779842615127563, 0.5928186774253845, -1.3029481172561646, 0.8744061589241028, 0.6186076402664185, -1.4467519521713257, -1.508078694343567, 0.9004617929458618, -0.39565449953079224, 0.8866216540336609, 0.39889252185821533, -0.0021672395523637533, -0.8572186827659607, 1.309571385383606, 0.7347307205200195, 0.36930787563323975, 0.7752454280853271, -0.6442700624465942, 0.19796137511730194, 0.8469183444976807, -2.0270729064941406, -0.5077933073043823, -0.15180279314517975, 2.037963628768921, 0.32792752981185913, -0.6407516002655029, -0.9661300182342529, 0.6022636294364929, 0.7378243803977966, 0.5477393269538879, 0.0898904874920845, -0.35493314266204834, 0.435333251953125, -2.577427387237549, -1.0648057460784912, -0.4433783292770386, -0.7882660627365112, 1.3468612432479858, -0.09531159698963165, -0.15689963102340698, 0.8705031871795654, -0.934863805770874, 1.4260568618774414, 0.22144246101379395, -0.7113659977912903, 0.4911983609199524, 1.7099800109863281, -0.22038838267326355, -0.4660089612007141, 0.4461776316165924, 1.3615068197250366, -1.0933263301849365, -0.04609697312116623, 0.39222392439842224, 0.14883758127689362, -0.7802436351776123, -2.1424782276153564, -0.6657883524894714, 2.0848844051361084, -0.3136413097381592, -0.28612789511680603, 0.24369530379772186, -1.37177574634552, -1.1993167400360107, -0.47063589096069336, 0.4578815698623657, -0.8517670035362244, 2.1122660636901855, -0.6572784781455994, 0.06920955330133438, 0.6449477076530457, 0.6706510782241821, 0.5110860466957092, 0.9628563523292542, -0.6118696331977844, 1.206983208656311, 1.129193663597107, -0.46789517998695374, 0.6500265598297119, -1.2469491958618164, 1.1922780275344849, -0.21984604001045227, -0.49871811270713806, -0.40214821696281433, 0.569805383682251, 0.3810049593448639, -0.34494590759277344, -0.6287805438041687, 0.20336127281188965, -0.8340557217597961, -0.10527370125055313, 0.5110757350921631, 0.6630226969718933, -0.23773831129074097, -0.5239238142967224, -0.084889255464077, 1.1912801265716553, -0.753553032875061, -1.6691511869430542, -0.29947012662887573, 0.21095049381256104, -0.37311360239982605, -0.6306449174880981, -0.3650510311126709, -0.3944602906703949], [-1.2827551364898682, -0.02992967888712883, 0.05089765787124634, 1.1612458229064941, -0.08108023554086685, -0.6549656391143799, 0.77921462059021, 0.580041229724884, -0.3564072251319885, -0.9494184255599976, 1.4280343055725098, 0.8797385692596436, -0.21499621868133545, 0.32148709893226624, 0.01775936409831047, 1.6686365604400635, -0.9378952980041504, -0.17384205758571625, 0.3809281289577484, -0.05059226229786873, -1.3438631296157837, -0.6924619078636169, 1.2024239301681519, 0.7402504682540894, -1.3058373928070068, -0.357053279876709, -1.5738054513931274, 1.7390973567962646, -1.0933727025985718, 0.2281464785337448, -1.330065369606018, 0.05074700340628624, 0.031596627086400986, 0.8389782905578613, 0.5919058918952942, 0.28417500853538513, 1.5779078006744385, -0.3713873624801636, 0.30067965388298035, -0.7191270589828491, 0.8129401803016663, 1.1875035762786865, 0.6601062417030334, -0.22343043982982635, -1.215376615524292, -0.5534751415252686, 0.5517446994781494, 1.9379830360412598, 1.707374930381775, 0.4423973560333252, 1.1110516786575317, -1.0207921266555786, -1.0356342792510986, 0.0411851704120636, 0.5163069367408752, 0.1717807948589325, -0.8165566921234131, 0.2873932123184204, -0.48770803213119507, 0.2542612850666046, 1.412692666053772, -0.622431755065918, -0.13704785704612732, 0.12898044288158417, -2.112407922744751, 0.4575789272785187, 1.4327789545059204, 1.2216705083847046, 0.49773985147476196, 1.0264981985092163, -0.829365611076355, 1.1045160293579102, -1.1995857954025269, -0.5053980946540833, 0.14143367111682892, 0.5220898389816284, -1.4262059926986694, 0.2602348029613495, -0.16637587547302246, 0.07633060216903687, 0.07303453236818314, -1.6438956260681152, 0.21679270267486572, -0.2892499268054962, -0.611011803150177, 0.5182179808616638, -0.3983369469642639, 0.07801661640405655, -2.1025309562683105, -0.4551553726196289, -0.13411961495876312, 0.03897172212600708, -0.28951799869537354, 0.39088186621665955, -0.8930699229240417, 0.5757213830947876, 0.2852547764778137, 0.4059082865715027, -2.655357837677002, 0.9735462069511414, 1.1142786741256714, -0.48647260665893555, 1.0420606136322021, 0.5052632093429565, -1.2605682611465454, -0.5810598134994507, 0.22074195742607117, 0.5255963206291199, -1.380910873413086, -0.18955878913402557, 0.7348366975784302, -0.11783193796873093, -1.1134752035140991, 1.0270907878875732, 0.9435257315635681, 0.8233851790428162, 0.7715675830841064, 0.00741865299642086, -0.558370053768158, 0.1834477037191391, 1.5041959285736084, -0.5131657123565674, 0.868557333946228, -0.6394776701927185, 0.2504895031452179, 0.09265708923339844, -0.8773276209831238, 0.0872548297047615], [0.03828778490424156, -0.9508213996887207, -1.2995392084121704, 0.22270043194293976, 0.47350040078163147, 0.806120753288269, -0.3079897463321686, -0.9532163143157959, -0.9153025150299072, 0.06654731929302216, 0.8755107522010803, 0.1927158683538437, -0.1421239823102951, 0.6401898264884949, -2.054140567779541, -0.43962714076042175, 1.0295435190200806, 1.549242615699768, 1.6454323530197144, -0.11195401847362518, -0.535382091999054, 0.3177790343761444, 1.0613725185394287, -1.9181809425354004, -0.09108866751194, -0.21387790143489838, 0.7742493748664856, 0.33941730856895447, -1.630549430847168, 1.088186264038086, -0.8070424199104309, 0.4171966314315796, 0.13510309159755707, -0.567460834980011, -0.08806783705949783, -0.792441725730896, 0.515986442565918, 2.88430118560791, -1.3479341268539429, -0.28087863326072693, -1.0139145851135254, -0.8014268279075623, 0.3930438160896301, 0.25054648518562317, -0.7581112384796143, 1.7421170473098755, 0.5916699171066284, 0.34603890776634216, -0.4858746826648712, 1.2622190713882446, 0.26157477498054504, -0.9038310050964355, -0.2869124114513397, 1.3348116874694824, 0.38342148065567017, -2.323953628540039, -0.47173380851745605, 0.6901111006736755, -1.4964070320129395, -0.14116637408733368, 0.39246445894241333, -1.6435574293136597, -1.068893551826477, 0.7517480850219727, 3.4166765213012695, 1.3116400241851807, 0.7012590169906616, -1.0142502784729004, 0.9054021239280701, -0.7285762429237366, 0.18331637978553772, 0.3847658038139343, 1.4246842861175537, -0.5369272828102112, 0.6724828481674194, 0.12267837673425674, 0.41466763615608215, -1.9681988954544067, -0.873186469078064, 0.30062583088874817, 0.9559704065322876, 1.8153316974639893, 0.27066487073898315, -2.1646881103515625, -0.5999767184257507, 0.8445261716842651, 1.3639742136001587, 0.3752429187297821, 0.7114813923835754, 0.1373777687549591, 0.008916775695979595, -0.21350568532943726, -0.5785935521125793, 0.45949453115463257, 0.899236798286438, 1.0434353351593018, 0.30567875504493713, -0.23865176737308502, -0.5527057647705078, 0.5492702126502991, 0.06132066622376442, -0.23427239060401917, 1.0449527502059937, -0.294847309589386, 0.2533269226551056, -0.46256890892982483, -0.10855463892221451, 0.7983354330062866, -0.179287850856781, 0.5867663025856018, 1.3228967189788818, -1.2227433919906616, -0.8629857301712036, -0.03039383888244629, -1.0980981588363647, -0.5022823810577393, -0.9877751469612122, 0.4412199854850769, 0.39107924699783325, 0.3437992036342621, 0.08867784589529037, -1.2653340101242065, -1.7797574996948242, -1.745208740234375, -1.4031705856323242, 1.245587706565857, 0.31509771943092346, 1.7287100553512573], [-0.6804041266441345, 0.32543137669563293, -0.8097848296165466, -0.2625347077846527, -0.11196092516183853, -0.010372907854616642, -0.9308649897575378, -0.5845635533332825, 0.7393662929534912, -0.28516143560409546, 1.6202408075332642, 1.8508522510528564, 0.9308972954750061, -0.03198263421654701, 0.6467229127883911, 0.19704805314540863, -1.6728322505950928, -1.1924382448196411, 0.5965171456336975, -0.5088545680046082, 1.030879020690918, 0.48393556475639343, 1.090716004371643, 0.07221375405788422, -0.7821584939956665, -0.6323122978210449, 0.9181134700775146, -0.5220168232917786, 0.6924683451652527, 0.17223255336284637, 0.2607109844684601, 0.1665390133857727, -0.8422961235046387, 0.20011650025844574, 1.1752954721450806, -0.3564068078994751, -0.06615413725376129, -0.38353899121284485, -1.5351626873016357, 0.33776339888572693, -1.396956205368042, 0.07204543799161911, 2.001378059387207, 1.3377587795257568, -1.2690250873565674, -1.3764876127243042, -2.523134469985962, 1.5678527355194092, 0.022357482463121414, 0.5982815623283386, -2.543987274169922, 0.7847080230712891, -0.44085219502449036, -0.5021742582321167, 0.8161281943321228, 1.1990259885787964, 2.067660093307495, -0.7969453930854797, -0.537634551525116, -0.5783528089523315, -0.8449276089668274, 2.2013134956359863, -1.5675312280654907, -0.18873783946037292, 1.2435065507888794, 0.37158674001693726, -2.107945680618286, 1.3472223281860352, -0.034413255751132965, 0.21857289969921112, -0.15371982753276825, 2.550153970718384, 0.195419579744339, -0.21005231142044067, 0.7421029806137085, 0.008736246265470982, 0.29887154698371887, -0.32347166538238525, -1.4655094146728516, 0.08949923515319824, -2.0299315452575684, -0.6478903889656067, 0.8664385676383972, 0.4156845808029175, -0.2705937922000885, 0.1490640640258789, 0.0345722995698452, 0.6305489540100098, -0.051897987723350525, 0.6886022090911865, -0.01762746460735798, 0.3563857674598694, 1.7360427379608154, 0.12435146421194077, 0.8894402384757996, -0.3595265746116638, -0.3869439363479614, -1.3643332719802856, 0.7445081472396851, -0.8464866876602173, 0.2103521078824997, 1.42634117603302, 0.3599783182144165, -0.3803669512271881, -2.757962703704834, 0.49653613567352295, 0.40886232256889343, -0.37911951541900635, 1.6868515014648438, 0.20675596594810486, 1.219359278678894, 0.9187009334564209, 1.0834165811538696, 0.20812396705150604, -1.3644814491271973, -0.21914665400981903, -1.3684414625167847, 0.8825092911720276, 0.4284112751483917, 1.4069007635116577, 0.4152478575706482, 0.9993048310279846, -1.1435929536819458, 0.6340488791465759, -0.23874402046203613, 0.258809357881546, -0.7301089763641357, 0.6146978735923767], [0.6370798349380493, -1.8175313472747803, 0.4866507947444916, -0.4459938108921051, -1.1918758153915405, -0.931973397731781, 0.0882231816649437, -0.41277772188186646, -1.843024492263794, -0.9924567341804504, 0.07062900811433792, 0.2247147411108017, 0.35762178897857666, -1.909929633140564, 0.7326352596282959, -0.6059151887893677, 0.3342778980731964, -0.14920099079608917, 0.6588447093963623, 2.4969496726989746, 1.5419749021530151, 0.6110708713531494, 0.5731682777404785, -0.7112318277359009, 0.5133885145187378, 0.3296060562133789, 0.07241465896368027, 0.8787316083908081, -0.9162952899932861, -0.05688738450407982, -1.0693997144699097, 0.5421831607818604, 0.8597774505615234, -0.7195088267326355, 1.2703003883361816, -1.0121532678604126, 0.6198654174804688, -0.18986566364765167, -2.0422558784484863, -0.9500692486763, -0.6930499076843262, -0.41666844487190247, -0.38937029242515564, -0.6950388550758362, -0.024379340931773186, 0.341705322265625, -0.05128377303481102, 1.3440028429031372, -1.1359716653823853, -0.6781301498413086, -0.3203122019767761, 0.22897261381149292, 0.5910891890525818, -2.8280599117279053, -0.9190247058868408, -1.0759814977645874, 0.2322973906993866, 0.9935697913169861, 1.597041368484497, -0.057927269488573074, 0.299381822347641, 0.16832590103149414, -0.7263746857643127, -0.6322574019432068, -1.3056849241256714, -0.2046869397163391, 0.4919046461582184, 0.305149108171463, 1.0636324882507324, -0.4677942395210266, -1.4681183099746704, -0.6432753801345825, 1.5414315462112427, 0.8157193660736084, 0.23543347418308258, -0.28471216559410095, 0.9748085141181946, -0.7578650116920471, 0.7541871666908264, 0.12106936424970627, 1.5163905620574951, 0.5406962037086487, 0.1412438601255417, 0.051450494676828384, -0.7637428045272827, -0.040543634444475174, 0.7013237476348877, 3.0129942893981934, 0.16165365278720856, 0.8037144541740417, 0.06780027598142624, -0.4991644024848938, 0.7696542143821716, 0.5430051684379578, -0.31404370069503784, -0.742429792881012, -1.6938258409500122, -0.007797385565936565, 1.9733856916427612, 0.47398263216018677, -0.24551647901535034, 1.2206404209136963, 0.17014464735984802, -0.9091746807098389, 0.37733393907546997, -0.17730778455734253, -2.4966773986816406, 0.9482818245887756, 0.4135868549346924, 0.655765950679779, -2.5437169075012207, 2.0704336166381836, 1.7166374921798706, 0.4579642117023468, 0.08620196580886841, 0.7645164728164673, 1.0930081605911255, 3.0603394508361816, -0.031854838132858276, 0.03343501314520836, 0.46627694368362427, -1.8721681833267212, -0.33309459686279297, 0.175310879945755, -0.0003743361448869109, -1.4572694301605225, -1.9298834800720215, 1.3666001558303833], [0.7437989711761475, -0.11354796588420868, 0.0419461727142334, 0.45738092064857483, -0.8771273493766785, -0.3529185950756073, -0.20646575093269348, 0.5325681567192078, 0.6618662476539612, 0.2253981977701187, -0.9854363799095154, 0.028728915378451347, 2.392008066177368, -0.009962466545403004, 0.4481569230556488, -0.39876729249954224, 1.8127005100250244, -0.1630454808473587, -2.5617151260375977, -0.4438408315181732, 0.9628298878669739, -1.0855777263641357, -0.27414408326148987, -0.10576590150594711, -0.09653809666633606, -1.2536990642547607, 1.411331295967102, 0.08712618052959442, 0.9314472079277039, 2.5981454849243164, 0.04560735449194908, 0.7655969262123108, 0.9933832883834839, 1.6218262910842896, 1.4069499969482422, 0.013059000484645367, -0.4103282392024994, -0.6225892901420593, -0.2936415672302246, -0.6561352610588074, 0.23286974430084229, 1.6199243068695068, 0.008239404298365116, 0.9035786390304565, 0.8447625041007996, -1.1329914331436157, 2.726304054260254, -0.6612989902496338, 0.009477206505835056, -0.4258006513118744, 0.0838417336344719, 0.7376950979232788, -1.9967632293701172, 1.1196690797805786, 0.1294933557510376, -0.48285815119743347, -1.2667306661605835, -0.7519830465316772, -0.8827942609786987, -0.11870308220386505, -1.3207824230194092, 1.6303203105926514, -1.0404466390609741, -1.6208592653274536, -0.48972561955451965, 0.6462391018867493, -0.3989124298095703, 0.3885112404823303, -2.7828776836395264, -1.5476703643798828, 0.14597147703170776, -0.12547902762889862, -0.1985716074705124, -1.0518156290054321, -0.1378738433122635, -0.6795271635055542, -1.0963051319122314, 1.6076483726501465, 0.31564071774482727, 1.3624651432037354, 0.12210449576377869, -0.7130804061889648, 0.9804190397262573, 1.0243630409240723, 0.26428651809692383, -1.0064308643341064, 1.3100061416625977, 0.7437922358512878, 0.5761165618896484, 0.5111591815948486, 0.3560100793838501, 1.7211921215057373, -0.32408371567726135, -0.7470819354057312, 0.7847375869750977, 0.7146501541137695, 3.4964256286621094, 0.8392844200134277, 1.2358161211013794, 1.7715959548950195, -0.15206769108772278, 0.19129255414009094, 1.2131093740463257, 1.0929614305496216, 0.7151090502738953, -0.3510848581790924, 0.586783230304718, -2.1884827613830566, -0.7684250473976135, 1.2011157274246216, -2.2720987796783447, -0.8486828207969666, 0.32294437289237976, -0.5454268455505371, 0.2567129135131836, -0.3603636920452118, -0.3754519522190094, -1.8147602081298828, -0.3700697720050812, -0.11214817315340042, -0.810030460357666, -0.962097704410553, 0.8730619549751282, 0.07202287018299103, -0.8983675241470337, -0.16203507781028748, -0.5217795372009277, -0.7404146194458008], [-0.2973375618457794, -0.03520958125591278, -0.25169673562049866, -2.226073980331421, -0.46091949939727783, -0.03855069354176521, -0.37723687291145325, 1.0587972402572632, -0.17032790184020996, -0.6458597183227539, 0.5986124277114868, 0.4216858446598053, -0.262029230594635, 0.22805078327655792, -0.17575131356716156, -0.1624567061662674, -0.07993044704198837, -1.8091753721237183, 1.4067790508270264, 0.8706493377685547, -0.7598238587379456, -0.1848057508468628, 0.30662915110588074, 0.9875702857971191, 0.9899836778640747, 0.8325196504592896, -0.2065257430076599, -0.9134083986282349, 1.2892733812332153, 0.9904146194458008, -0.28139445185661316, -1.7334169149398804, 0.25898993015289307, -0.011488315649330616, -0.0036527153570204973, 1.0765190124511719, 0.6425626873970032, 0.941593587398529, 0.09791690856218338, -1.3047226667404175, 1.0308507680892944, 1.0098267793655396, -1.011330485343933, -0.8785686492919922, 2.0647335052490234, 1.7991771697998047, -0.6377159357070923, -0.779600977897644, -0.6112164855003357, 0.3498663306236267, -0.6983497738838196, 0.797257125377655, 0.9049500823020935, 1.2908316850662231, 0.9838460087776184, -1.3020676374435425, 0.7701166868209839, 0.3174744248390198, 1.8337593078613281, -2.0981922149658203, -1.2717832326889038, 1.0764241218566895, 0.5796056985855103, 0.7568079233169556, 0.06612030416727066, 0.5160870552062988, -0.39720356464385986, 0.1355503499507904, 1.1206223964691162, 1.0060153007507324, -1.3322649002075195, 0.4230049252510071, 0.9158532023429871, 0.7442010045051575, 0.09458135068416595, 1.2459861040115356, 0.6094084978103638, 0.5657076835632324, 0.6666818857192993, -1.0327073335647583, 0.13863520324230194, -0.6747426986694336, 0.7459639310836792, -0.2979833781719208, 0.49033159017562866, 0.5708236694335938, -1.8335421085357666, -0.15970827639102936, -0.863289475440979, 1.628622055053711, -1.4727400541305542, 0.08143415302038193, 1.3412407636642456, -0.5341415405273438, -1.9739294052124023, -0.7104476690292358, 0.37574219703674316, -1.2930364608764648, -0.519273579120636, -0.4570220708847046, 2.126234769821167, -0.4720793664455414, 0.3250398635864258, -1.4588121175765991, 0.19195079803466797, 0.28009992837905884, 0.6534373164176941, -0.40708446502685547, 1.3748987913131714, 0.31221887469291687, 1.2764166593551636, 0.2432786077260971, 1.036885380744934, -0.537813663482666, 1.509490728378296, 0.20689094066619873, 0.661693274974823, -1.8539589643478394, -0.11862587183713913, 1.1845636367797852, -0.8768914341926575, 0.5390234589576721, 0.10566147416830063, 0.8065757155418396, 0.9534291625022888, -0.9318389296531677, 2.7964017391204834, -1.2194651365280151], [1.1972382068634033, 1.2890931367874146, 1.4600884914398193, 1.4476916790008545, 1.8165570497512817, -1.4541789293289185, 1.4547526836395264, 1.5829696655273438, 0.07956359535455704, 0.49322953820228577, -0.10586701333522797, 0.1800125688314438, -0.6522083282470703, -1.7719415426254272, -0.42458251118659973, -0.7219791412353516, 0.8604075312614441, -0.8763561844825745, -0.19982078671455383, 0.11107586324214935, -0.4448302388191223, 0.643783450126648, -0.05956358090043068, 1.341058611869812, 0.5735312104225159, -0.92320317029953, -0.9911594986915588, 0.41018664836883545, 0.028551287949085236, -1.0041338205337524, 0.8278215527534485, -2.659942865371704, -1.269814133644104, -1.2626694440841675, 0.7048355340957642, 1.0943849086761475, 0.17072328925132751, -0.66692715883255, -0.24847422540187836, -0.3470500409603119, -1.7448773384094238, -0.40076032280921936, 0.19969722628593445, 0.5535138845443726, -0.5358095765113831, 0.3013915419578552, -0.5723943710327148, -0.67413729429245, -0.5597115159034729, 0.625575840473175, -0.22335518896579742, -0.43173107504844666, -0.6867623925209045, 1.649782419204712, 1.1778022050857544, -0.8073751926422119, 0.008268672041594982, 0.5199518799781799, -0.5037801861763, 0.7015950083732605, 3.0762674808502197, -1.1479148864746094, 0.9050209522247314, -0.5555834174156189, -0.3985148072242737, 1.1662451028823853, -0.6441691517829895, -0.18717263638973236, -1.8960225582122803, -0.9784349799156189, -0.3150501549243927, -0.32310810685157776, 4.6749024477321655e-05, 1.5355875492095947, 2.0311121940612793, -1.2737668752670288, -0.3872087001800537, -0.1629868894815445, 0.27913087606430054, 0.509772002696991, -1.0635608434677124, 0.9022485017776489, 0.8615485429763794, -0.7230330109596252, 0.5403497815132141, 0.4419559836387634, -0.18041642010211945, -0.8467296957969666, 0.47420772910118103, 0.026039352640509605, 0.4003094434738159, -0.3386659324169159, 0.5586936473846436, 0.714949369430542, 0.17962470650672913, 1.473393440246582, -1.5235178470611572, -0.1384398490190506, 0.7739225625991821, 0.6298938393592834, 1.1335933208465576, -0.3477800786495209, -0.2418559193611145, 1.2988090515136719, -1.1018973588943481, 0.8590473532676697, 1.4544603824615479, 0.12161213904619217, -0.17034879326820374, 1.836063027381897, 0.6642548441886902, -0.48746493458747864, -0.7027187347412109, 0.4348240792751312, 0.011033877730369568, -0.862519383430481, -0.1955001950263977, 1.780346155166626, 0.8505296111106873, 1.7411160469055176, 0.27023807168006897, -0.46932709217071533, 0.5451105833053589, -1.6123073101043701, -0.42531970143318176, 0.5096415281295776, -1.373806118965149, 1.3286608457565308], [-0.4601983428001404, 1.7157186269760132, -1.3532977104187012, 0.6089658737182617, -0.29084616899490356, 0.6411770582199097, -0.15236423909664154, 0.12855572998523712, -1.7384709119796753, 0.16501088440418243, 0.4166755676269531, -0.5243573784828186, -0.38458701968193054, -0.32148826122283936, -0.20999230444431305, 1.4550927877426147, -0.3287866413593292, -0.46057385206222534, -0.7600634098052979, 0.4715062081813812, -0.4480604827404022, 0.5851181745529175, 1.8998874425888062, 1.1857274770736694, 0.30669838190078735, 0.6775880455970764, -0.035732343792915344, -1.6867743730545044, -0.1582910418510437, 0.5357309579849243, 0.06536245346069336, 0.6663979291915894, 0.20198389887809753, 0.5457658767700195, 0.8687990307807922, 0.7622513771057129, -0.47897276282310486, 0.05854896456003189, -0.17325223982334137, -1.714684009552002, 0.6459184885025024, 0.31468892097473145, 0.024837344884872437, -1.439274787902832, -0.28910428285598755, -2.0089194774627686, -0.15019771456718445, -1.1608716249465942, 1.6835986375808716, -0.0020766332745552063, 1.1450129747390747, 0.5990259647369385, 0.5507984757423401, -2.4003639221191406, 0.28478989005088806, -0.5277193188667297, -1.2960797548294067, -0.11256616562604904, -1.480505347251892, 0.8824741840362549, -0.20068225264549255, -1.5477385520935059, -0.8290485739707947, -1.0784376859664917, 0.03955969214439392, -0.9569100141525269, -0.14494365453720093, 0.5139186382293701, -0.11905259639024734, -0.6618125438690186, -2.211766481399536, -0.8602185249328613, -0.7449347376823425, -0.05111793056130409, 1.082645297050476, 0.41104739904403687, -0.2852638363838196, 0.7694353461265564, -2.21002197265625, 0.32769182324409485, -2.3358051776885986, -0.8418422341346741, -0.6546998023986816, -0.2960958778858185, 0.9977360367774963, 1.2056033611297607, -0.1259913593530655, 1.2595431804656982, 0.22420762479305267, 1.1988083124160767, -0.25338035821914673, -0.03225493058562279, -1.1751959323883057, 0.3201199471950531, 0.8747539520263672, -0.6656374931335449, 1.1283890008926392, 0.5354143977165222, 0.02175885997712612, 0.36391162872314453, -0.9363462328910828, 0.16021206974983215, -1.3691115379333496, -0.02601691521704197, -0.8402562737464905, 0.3292466998100281, -0.401755154132843, 0.19514240324497223, 0.3794945776462555, -1.1823015213012695, 0.5515711307525635, -0.9475882053375244, -0.025660617277026176, -0.4755396544933319, -1.4348434209823608, -2.507049798965454, 0.7734730839729309, -0.05070880800485611, -0.9522860050201416, -0.7403756976127625, 0.930871844291687, 1.3613991737365723, 1.20717453956604, 0.5031662583351135, -0.31059736013412476, 0.79571133852005, 0.35082727670669556, -2.3697495460510254], [-1.509514331817627, -0.3689460754394531, 0.09042089432477951, -0.555568516254425, -0.2576932907104492, -0.8847155570983887, 2.9286916255950928, -0.4520455002784729, 0.3027117848396301, -0.26713865995407104, -0.20667394995689392, -0.7598147392272949, 1.8941055536270142, 1.9831753969192505, -0.08819147199392319, 0.35756373405456543, 1.3090553283691406, -1.4050770998001099, 1.0417227745056152, 3.0171241760253906, -2.1290154457092285, -0.29754650592803955, 0.0737004354596138, 0.7057107090950012, -0.39648008346557617, -0.4584657549858093, -0.34565243124961853, 0.23537614941596985, -0.49960950016975403, -0.6163718700408936, 0.8890163898468018, -0.7320231199264526, 0.5352514386177063, 1.0821459293365479, 1.3708759546279907, -0.4809257388114929, 0.15147916972637177, 0.5356667041778564, -0.04423738270998001, 1.5535062551498413, -0.6260584592819214, 1.9625599384307861, -0.5701040625572205, -1.2522425651550293, 1.54082190990448, 0.8367566466331482, 0.4642186462879181, -0.812471330165863, -1.3354471921920776, -0.4175850450992584, -0.7794025540351868, 2.0141444206237793, -0.6150277256965637, -0.5828329920768738, 0.42438527941703796, -0.29900670051574707, 1.4327083826065063, 0.6560462117195129, 0.6537211537361145, -0.019052837044000626, -0.8632040619850159, -0.489680677652359, -1.3620020151138306, -0.20899352431297302, 0.8207676410675049, 1.721649408340454, -1.2266221046447754, 0.7253512144088745, 0.2661908268928528, 1.2153126001358032, -1.3771545886993408, -0.2272852063179016, -0.16784781217575073, 0.5984860062599182, -1.060454249382019, -0.5612750053405762, -0.4852268695831299, -1.2575761079788208, 0.6012894511222839, 1.4424694776535034, -0.12845751643180847, -2.0034468173980713, -2.3753855228424072, -0.4509451687335968, -0.37471604347229004, -2.0395779609680176, 0.9667499661445618, -0.12095074355602264, -0.8343600630760193, 1.2495551109313965, 0.041892699897289276, -0.9806046485900879, 0.186080202460289, 0.009042922407388687, -1.8575905561447144, 0.42875808477401733, -0.15565375983715057, 0.4866628348827362, 0.574647843837738, -1.1049152612686157, 0.9519853591918945, -1.1964248418807983, 1.5875937938690186, 0.6277761459350586, -1.3258345127105713, 0.6033731698989868, 0.35229718685150146, 1.311478853225708, -0.975641131401062, 0.6524600982666016, 0.34381014108657837, 0.7070921659469604, 1.5109519958496094, -0.02721489407122135, -0.051606230437755585, 0.43960797786712646, 0.19175903499126434, -0.7600454688072205, 0.8472272753715515, -0.4264480173587799, 0.6574605107307434, -1.0874749422073364, -1.71307373046875, -3.822582721710205, -0.8466974496841431, 0.18764185905456543, -0.1621733158826828, -0.22059696912765503], [-0.031224366277456284, -0.37512239813804626, 0.2147614061832428, 0.3918687403202057, -2.0718884468078613, -0.131989985704422, -0.3569897711277008, 0.6767802834510803, -1.2599647045135498, -0.6903677582740784, 0.07260338962078094, -0.7590836882591248, -0.2923429012298584, -0.4418477416038513, 0.1371639370918274, 0.923584520816803, 1.690759539604187, -0.7302415370941162, 0.28174149990081787, 1.402806282043457, 1.0206514596939087, -0.8412641882896423, 1.1691207885742188, 0.7467790842056274, -0.21547052264213562, -0.9501984119415283, -0.3401239812374115, -1.104145884513855, 0.9449641108512878, -0.22734591364860535, 0.41280290484428406, 0.8684067130088806, -0.4291398525238037, 0.626957893371582, -0.2044883370399475, -0.7345702648162842, -0.19247041642665863, 0.9184908270835876, 0.8078892827033997, -1.4662624597549438, 0.7124481797218323, -1.7534233331680298, -0.536489725112915, 0.5487954616546631, -0.6347485184669495, -1.9488621950149536, 0.7012184262275696, 1.2927955389022827, -0.334859162569046, -0.18943895399570465, 1.363416314125061, -0.2675037384033203, -0.6786037683486938, -0.23262692987918854, 0.5666512846946716, 2.0177712440490723, -0.1938718855381012, -0.03372958302497864, 0.9269269704818726, -2.1450650691986084, -1.4221441745758057, -1.445163607597351, -1.8013813495635986, 0.9290367364883423, -0.44324758648872375, -0.8563188314437866, 1.0484317541122437, 1.3440825939178467, 0.6099178791046143, 0.42929646372795105, 0.008121746592223644, -1.773158073425293, -1.8516823053359985, 1.4949692487716675, 1.2555629014968872, 0.10617661476135254, 1.4615790843963623, 0.7503129839897156, -0.7047680020332336, -0.528876781463623, -0.24762719869613647, 2.0736281871795654, -0.4652829170227051, -1.379085898399353, 0.4813919961452484, -0.12950672209262848, 1.327870488166809, -0.5761699080467224, 0.9930869340896606, -0.42351001501083374, -0.0641326904296875, 0.024122120812535286, -0.5828508734703064, 0.5602450966835022, -0.4576125144958496, -2.1217234134674072, 0.6758357882499695, 0.0857100561261177, 0.7473993301391602, -1.7055047750473022, -0.33519428968429565, -1.118340015411377, -0.41327103972435, -1.9153969287872314, 0.4991757571697235, 0.4030824601650238, -2.2603108882904053, 0.8762795329093933, -1.867984414100647, -0.8017050623893738, 1.3633116483688354, -0.29809045791625977, 0.5634652376174927, -0.59776371717453, -2.9104163646698, -0.26575347781181335, -0.5787888169288635, 1.0910134315490723, 1.8653596639633179, 0.2005317509174347, -0.23143444955348969, 0.0426105335354805, -0.6682046055793762, -0.40353503823280334, 0.899351179599762, 1.6598745584487915, -0.9025594592094421, -1.4872374534606934], [0.525827944278717, -0.9507789015769958, 0.8642174601554871, -0.08060600608587265, -0.6877934336662292, -0.33291521668434143, 0.3022500276565552, 1.5593560934066772, -0.9748132228851318, 0.2680862247943878, -0.561438798904419, -0.26188966631889343, 0.6134961247444153, -0.07123435288667679, -1.157068133354187, 0.4851830005645752, -0.19474251568317413, 1.0694944858551025, 1.1699258089065552, 1.5046881437301636, 0.03350881114602089, -0.3241189420223236, -0.6481562256813049, 0.8815291523933411, 0.7911455035209656, 0.6040866374969482, 1.3136539459228516, 0.8492633700370789, 0.2528519332408905, 0.45811790227890015, 0.8949987292289734, 0.002756438683718443, -1.0661405324935913, 0.01403167936950922, -1.2960211038589478, 2.4983859062194824, -0.7000892162322998, -0.07042161375284195, 0.6775902509689331, -1.1675208806991577, -0.7037816643714905, 2.9634666442871094, -0.48503121733665466, -1.049306869506836, 0.3298165798187256, 0.7584425806999207, 0.17096960544586182, -0.18505828082561493, 0.6706385016441345, 1.0235685110092163, -0.39029937982559204, 0.8549193739891052, 1.0046533346176147, -0.08707369118928909, -0.9341357350349426, -1.7298941612243652, 0.1700647473335266, 1.3064165115356445, 1.4078285694122314, 0.9677630066871643, -0.7378663420677185, -1.0939875841140747, -0.5764639377593994, -0.5374088287353516, -0.09007348865270615, 0.37728723883628845, -0.8028473258018494, -0.3217352330684662, 0.023508034646511078, -0.25679004192352295, 1.3141449689865112, 2.0478482246398926, 0.16387486457824707, 1.3294172286987305, -0.43853262066841125, 1.2618190050125122, 0.5368868708610535, -0.9902260303497314, 0.3177661895751953, -1.1885998249053955, 0.534489095211029, -1.8762072324752808, 0.017634300515055656, -3.310772657394409, -0.3976391851902008, -0.6800106763839722, 0.5059414505958557, -0.14612977206707, -1.3112032413482666, -0.9789907336235046, -0.0902879387140274, -0.2772059440612793, -0.3157809376716614, 0.3716847002506256, -1.1778032779693604, -1.2756050825119019, 0.18373413383960724, -0.6131279468536377, 0.08814098685979843, 1.023133397102356, 0.18962466716766357, 1.5484727621078491, -0.6530538201332092, 0.4596259891986847, 0.7269070744514465, -1.7588850259780884, -0.9338526725769043, -0.5058485269546509, 1.525684118270874, 1.0695053339004517, 2.0898165702819824, -1.5814406871795654, -0.132808655500412, 0.799256443977356, 0.49112293124198914, 0.23373554646968842, -0.8586317896842957, -0.5852786302566528, -1.7061495780944824, -0.2197338044643402, 0.08561538904905319, 0.7933294177055359, 1.655752420425415, 1.0268710851669312, 0.09883073717355728, 0.7001280784606934, -0.4386458396911621, 0.16569091379642487], [-1.074610948562622, -0.9978079795837402, -1.3206093311309814, -0.8641355633735657, 1.001049518585205, 0.7704272270202637, 0.09327897429466248, -0.45986419916152954, 0.6059487462043762, -0.7633399963378906, 1.2996056079864502, 1.5549129247665405, 0.8309623003005981, 0.20915082097053528, -1.6194493770599365, -0.23057471215724945, -1.6649060249328613, 1.1538569927215576, -0.27451565861701965, -0.4557006061077118, 0.8166406154632568, 2.438230514526367, 1.1650104522705078, -1.4461671113967896, 1.626636028289795, -0.5063916444778442, -0.5554666519165039, -1.992200255393982, -0.8937212228775024, 1.5001732110977173, -0.508720874786377, -1.98220694065094, -0.8021009564399719, 0.33387190103530884, -1.2575675249099731, -0.14445415139198303, 0.5634322762489319, -0.05922301113605499, 0.8144862055778503, 0.1308658868074417, -0.857760488986969, 1.791702389717102, -0.17210611701011658, -0.26895636320114136, -2.844498872756958, 0.9189245700836182, -0.40728071331977844, -0.6480641961097717, -0.6542546153068542, -1.0218706130981445, 1.889186143875122, 0.8721096515655518, 1.648963212966919, 1.3077311515808105, 1.4922056198120117, -0.18261784315109253, -1.4627829790115356, 0.4510452151298523, -0.6508822441101074, -1.1147805452346802, -0.4136531949043274, -1.8461743593215942, -1.66277015209198, 1.1480095386505127, -0.2572418749332428, 0.38334301114082336, 1.465349793434143, 1.9079461097717285, 0.19673879444599152, 0.40322020649909973, 0.0012075034901499748, 0.5248679518699646, -0.8677366971969604, -0.6855734586715698, -0.7158117890357971, -2.4921486377716064, 1.882541537284851, -1.2163059711456299, 0.2540597915649414, 0.5333380699157715, -0.7087024450302124, 0.2987653911113739, 0.812500536441803, -2.016685962677002, 0.9668771028518677, -1.1936367750167847, 1.842031717300415, -1.4482086896896362, 1.0925956964492798, -0.33817416429519653, 2.6372010707855225, 0.1448451429605484, 0.12772813439369202, 0.39485591650009155, 2.3052337169647217, -1.9929330348968506, -1.0233465433120728, -0.06347683072090149, 0.37859734892845154, -1.7362912893295288, 0.5670220851898193, 0.9039829969406128, 0.8014806509017944, 1.322049856185913, -1.1238648891448975, 0.3447321355342865, -0.47561734914779663, 0.06237802654504776, 0.7355695962905884, -0.758520781993866, -0.8805093169212341, -0.18512417376041412, 0.1101025938987732, 2.1131467819213867, 1.357417345046997, -1.1169946193695068, -0.3651115596294403, 0.4528239965438843, -1.848804235458374, 0.46256023645401, -0.7646980881690979, -0.9337769746780396, 0.7487106919288635, 0.7683749794960022, -0.18950553238391876, 0.9495747089385986, -1.1180553436279297, 0.3288296163082123], [-0.9196618795394897, -1.7287213802337646, -0.1491638422012329, -0.1958039253950119, 0.27861446142196655, -1.0219906568527222, 0.9374403953552246, -0.4287013113498688, -0.4337928295135498, 1.6036782264709473, 0.30024996399879456, 1.6817442178726196, 0.2449328899383545, 0.004066560883074999, -0.3556769788265228, 0.9807415008544922, -0.21658870577812195, 1.4146877527236938, -0.7840853333473206, -0.1385093331336975, 1.1448135375976562, 0.9546535611152649, 1.1583974361419678, 1.8666164875030518, 1.8531360626220703, -1.020470142364502, 0.731042206287384, -0.503017783164978, 0.8146718144416809, 1.0023231506347656, 0.4136436879634857, -0.7410396337509155, -1.3891202211380005, -1.3703279495239258, -0.4890788495540619, -0.288115918636322, -0.3956213593482971, 0.18793758749961853, -0.5704978108406067, -0.3636654317378998, -0.9219467043876648, 0.10671275854110718, -0.9151618480682373, -0.09264186024665833, -0.9261941313743591, 0.954663872718811, 0.7782430052757263, -0.257406085729599, -0.10744938999414444, 0.5471811294555664, 1.041268229484558, -0.0537288598716259, 0.5637459754943848, 0.4010971486568451, 1.5558128356933594, 0.007368507795035839, 0.32971304655075073, -1.060341715812683, 0.7591286301612854, -0.6194765567779541, 0.5217900276184082, 0.15591707825660706, -0.12489040195941925, -1.3011802434921265, -0.32867908477783203, -1.052351474761963, -0.881009042263031, 1.376927137374878, 1.2850862741470337, 1.5023349523544312, 1.3475674390792847, -0.7273058295249939, 0.8405656218528748, -0.304973840713501, -0.4816051125526428, 0.048206839710474014, 0.6340292096138, 2.002686023712158, -0.8285728693008423, -0.7096712589263916, 0.7820954918861389, 0.36485838890075684, 0.8861941695213318, -0.36793115735054016, 0.5888804197311401, 1.094019889831543, -2.0718724727630615, 1.2630013227462769, 0.8012531995773315, 0.28747156262397766, 0.07987529784440994, 0.14624322950839996, 0.018709300085902214, -0.04032731056213379, 0.7156181931495667, 1.8096014261245728, 0.828241765499115, 0.5129683613777161, 0.27573731541633606, -0.5339649319648743, 1.2118430137634277, -1.032893180847168, -0.6097216606140137, -0.20792385935783386, 0.5052871108055115, 2.2203619480133057, -0.33119192719459534, 0.2112504094839096, -0.9961045980453491, 2.395111560821533, 0.8441649079322815, 0.47665417194366455, -0.9419996738433838, -1.045236587524414, 0.502233624458313, 1.8182721138000488, 1.8088488578796387, -0.1524251252412796, 1.0109614133834839, -0.7552810311317444, 0.3282370865345001, -1.7919989824295044, -0.3553769290447235, 0.0726817324757576, 0.41168564558029175, -1.0478708744049072, -0.16749611496925354, 0.10823900997638702], [0.4681227505207062, 0.2904437482357025, -0.13278181850910187, 0.3466569483280182, -0.13861620426177979, 0.24188901484012604, 1.6180315017700195, -0.8640843033790588, -1.2333197593688965, -0.03942276909947395, 1.5342576503753662, 0.7805175185203552, -0.34704774618148804, -0.33204251527786255, 0.42560479044914246, -1.572103500366211, -1.9439547061920166, 1.7058491706848145, 2.273820400238037, 1.311975121498108, -0.4866578280925751, -1.7050808668136597, 0.2879011332988739, -1.7631640434265137, -0.8129501342773438, -1.4724828004837036, 2.6720569133758545, 0.09943576157093048, 1.432713508605957, 0.9759116768836975, -1.1419676542282104, -0.7607651948928833, 1.031643033027649, 0.7369300127029419, -0.39199453592300415, 0.9124220609664917, -1.4245661497116089, -1.219705581665039, -0.5305808186531067, 1.1178487539291382, 1.0483657121658325, -0.11335201561450958, -0.7913246750831604, 1.0598584413528442, -0.21518439054489136, -0.4097931385040283, 0.34335556626319885, -0.7854981422424316, 1.3345820903778076, 0.538414478302002, 0.6383141875267029, 1.0624184608459473, 0.6169366240501404, 0.20924142003059387, 0.8117552399635315, -0.11306212097406387, 1.7629905939102173, 0.24277201294898987, -0.9779278635978699, -1.6281206607818604, -0.2773127257823944, -1.6446037292480469, 0.3384285271167755, 1.2838109731674194, 0.9721166491508484, -1.543574571609497, 1.1634397506713867, -0.5766012668609619, -0.3229503035545349, 1.2721558809280396, -0.7957586646080017, -0.08542853593826294, -2.598947525024414, 0.4150645434856415, -0.6265376806259155, 0.24679520726203918, -0.9779201149940491, 1.475150227546692, 1.512573003768921, -1.3396459817886353, -0.8429805040359497, 0.41465359926223755, 0.623141884803772, -1.77153742313385, -0.6349992752075195, 0.8619882464408875, 0.6947504878044128, 0.8623543381690979, 0.5123451352119446, -0.6388125419616699, -0.45504602789878845, -0.5065708756446838, -0.06342645734548569, 1.2435001134872437, -0.5121187567710876, 1.3832166194915771, 0.3792792856693268, -1.5328044891357422, -0.5674404501914978, -0.010692579671740532, -0.4921151101589203, 0.24329215288162231, 1.5935437679290771, -0.3872714936733246, -1.1629348993301392, 0.24178552627563477, -0.2887722849845886, 0.8426342606544495, 0.05994049832224846, 1.7205615043640137, 0.37369975447654724, 0.5593997240066528, 1.4403884410858154, -0.6147626042366028, -0.08834714442491531, 1.4407093524932861, -0.8649987578392029, 0.4136406183242798, -1.5861616134643555, -1.6163251399993896, 0.44508299231529236, 0.700311541557312, 0.41052019596099854, -0.17570242285728455, 0.7127659916877747, 0.41152071952819824, -0.7601349949836731, 1.1913774013519287], [-0.6620993614196777, 1.6103888750076294, -0.6047072410583496, -0.982129693031311, 0.5691308975219727, -0.5986242294311523, -1.3268303871154785, -1.526434063911438, 0.5388636589050293, -1.9914478063583374, -0.7096996903419495, 1.4230061769485474, 0.06477317959070206, 1.9746880531311035, 1.6030887365341187, -0.6559116244316101, 0.9333539605140686, 0.6178748607635498, -2.1944119930267334, 0.03474392741918564, -0.19589467346668243, -1.4246292114257812, -0.7958099842071533, -0.9129156470298767, 0.3634471297264099, 0.5866894125938416, 0.6135447025299072, -1.148488998413086, -0.44474753737449646, -0.2011537402868271, -1.423324704170227, -0.5091969966888428, 0.47951817512512207, 1.4947497844696045, -1.1800709962844849, -0.06705302000045776, -0.2552317678928375, 0.5390483140945435, 0.42507249116897583, -1.390065312385559, -0.28968632221221924, 1.4958029985427856, 0.060761600732803345, 0.3650103509426117, 0.2752164602279663, -0.846846878528595, 0.6417477130889893, -0.9610976576805115, 1.2505298852920532, -1.2104997634887695, -0.776003360748291, -1.420397162437439, -1.5788599252700806, -0.5338925123214722, 0.22035925090312958, -1.1833621263504028, -0.7222675681114197, -0.3152472972869873, 1.0523808002471924, 0.01184858102351427, 0.9955674409866333, 0.6699885725975037, -0.15215526521205902, 0.8906643986701965, -1.3143651485443115, -0.6168785095214844, -0.0970817282795906, -0.17643727362155914, 0.4860825538635254, 0.38983967900276184, 0.12642721831798553, -0.36702871322631836, -0.8112855553627014, -0.5379928946495056, -0.596889853477478, -0.7681429386138916, -0.9594197869300842, -0.8772031664848328, -0.32761937379837036, 1.3977551460266113, -0.47018200159072876, -1.2260282039642334, -1.629944920539856, -0.9728997349739075, 2.3235080242156982, -0.40538179874420166, 0.7578083872795105, 0.06401252001523972, -0.635835587978363, 1.9792717695236206, -0.1326088160276413, 0.37652042508125305, 0.30122193694114685, -0.5177728533744812, -0.11043816804885864, -0.22665677964687347, 1.1464370489120483, -0.47183674573898315, -0.21708232164382935, 1.0109691619873047, -1.3298754692077637, -1.3502590656280518, -1.6063168048858643, -0.8886772394180298, 1.3857413530349731, -0.32797789573669434, 1.61770498752594, 0.08963152766227722, -0.25013765692710876, 0.48181042075157166, -1.2997053861618042, -0.583317220211029, 1.0135581493377686, 2.754211664199829, -0.36381959915161133, 1.9253484010696411, -0.5532252192497253, -0.19146883487701416, 0.024979671463370323, -0.3073577284812927, 1.7256793975830078, -0.034386783838272095, -0.0801912322640419, -0.41181039810180664, 0.9473317265510559, -2.330355167388916, -0.8681373000144958, -0.9728652238845825], [-0.3259774446487427, -0.28811123967170715, 2.2031054496765137, 0.6788741946220398, 0.47339603304862976, -0.6138579845428467, -1.5865973234176636, -0.8427490592002869, -1.082678198814392, 0.6705098748207092, 0.3660544455051422, 0.34398314356803894, -0.20865298807621002, 0.7715412974357605, 1.2476142644882202, -1.4019784927368164, 0.04082260653376579, -0.2818327248096466, -0.4530780613422394, 1.1814024448394775, 0.4092947542667389, 0.28805187344551086, -2.042891502380371, 0.38309037685394287, -0.2922815978527069, 1.5429794788360596, -0.0036659075412899256, -1.99934720993042, 0.5076224207878113, -1.0149173736572266, 2.0583581924438477, 0.36444127559661865, 0.6571493148803711, 1.0865509510040283, 1.2084977626800537, -0.14204348623752594, 0.44876715540885925, 0.9764418005943298, 1.1392929553985596, -0.0430060513317585, -0.4109119772911072, -0.4465409815311432, -0.9946406483650208, 0.6187107563018799, 0.016058219596743584, -0.09994454681873322, 0.07773828506469727, -1.4630142450332642, -0.3577705919742584, 0.15757757425308228, -0.7940547466278076, 0.04194119572639465, 0.14532381296157837, -0.18540741503238678, 0.17250218987464905, -2.20696759223938, 2.7030508518218994, -0.21454760432243347, -0.13624316453933716, 0.37690603733062744, 1.8549964427947998, -0.014904802665114403, -1.0481746196746826, -0.7470545172691345, 0.8225038647651672, 0.980626106262207, -0.2660464644432068, 0.19236716628074646, -1.1877528429031372, -1.2739449739456177, -1.0345749855041504, -0.6526355743408203, 2.146369218826294, -0.3056785464286804, -1.8505022525787354, -1.076988935470581, -0.39872753620147705, 0.7503153681755066, 0.21367645263671875, -0.26394814252853394, -0.8756751418113708, -1.1122108697891235, 1.651431679725647, -0.36723077297210693, -0.16030378639698029, -1.579978346824646, 1.7151850461959839, 1.2410987615585327, 0.8149164915084839, -0.19057703018188477, -0.11238245666027069, 0.5426734685897827, -1.153341293334961, -2.3246233463287354, -0.562593936920166, 1.3406352996826172, 0.34661275148391724, 0.7484866976737976, -0.906455397605896, -0.4224216043949127, 1.11231529712677, -0.1316179484128952, 0.6447846293449402, 0.42733466625213623, -0.18914379179477692, -0.6912724375724792, 1.0425071716308594, 0.03743595629930496, -2.399817705154419, -0.48975515365600586, -0.3477858901023865, 0.2104707658290863, 1.9453948736190796, 0.3605524003505707, 1.2465516328811646, 0.30850499868392944, 0.5177515149116516, 0.5995132327079773, -0.08219338208436966, -0.8767297863960266, 0.39485886693000793, -0.40338432788848877, -0.39120855927467346, 0.5315198302268982, -0.7794243693351746, 1.8042031526565552, 2.691391944885254, 0.22257056832313538], [1.9926981925964355, 1.4058023691177368, 2.968702554702759, -1.4811760187149048, -0.06027279421687126, -1.3678882122039795, 3.616220474243164, 1.4313658475875854, 0.15427491068840027, -1.907597303390503, 1.3508093357086182, 1.001157522201538, 0.4505631625652313, 1.177535891532898, 0.42879584431648254, 0.29957690834999084, 0.919662356376648, 0.01993997022509575, 0.17396418750286102, -2.066019296646118, 0.5814497470855713, 0.2966271936893463, 1.8086758852005005, -0.14828349649906158, -0.6262916326522827, -2.1030421257019043, 0.16642531752586365, -1.2335829734802246, -1.014328956604004, -1.2788199186325073, -1.8232566118240356, -0.4456404447555542, -1.0728346109390259, 0.4680439233779907, 0.4827134907245636, 0.4857160747051239, 0.35336822271347046, -0.062162235379219055, 0.9551058411598206, 0.9346866011619568, 3.7664406299591064, -0.8304390907287598, 0.5132840871810913, 0.47617971897125244, 1.392352819442749, -0.796371579170227, -0.05101092532277107, 0.14665308594703674, 1.585288166999817, 0.9604313969612122, -0.6496290564537048, 1.7413921356201172, -0.13061672449111938, -0.2731309235095978, 0.9965574741363525, -0.6325552463531494, -2.1317851543426514, -1.3929054737091064, -0.27955740690231323, -1.2652298212051392, 0.13704094290733337, 0.7083151340484619, -0.2177516371011734, 0.49729058146476746, 0.9896579384803772, -1.2923721075057983, -1.3024585247039795, 2.16127610206604, -0.04363070800900459, 1.3408035039901733, -1.044305682182312, 1.2618550062179565, -1.1780234575271606, 1.1332281827926636, -0.34109005331993103, -0.7557936310768127, -1.6834830045700073, 1.5766823291778564, 0.49034151434898376, -0.13243837654590607, -0.566685140132904, -0.260957270860672, -1.3315551280975342, -1.1857563257217407, -0.7350142002105713, 1.273310899734497, -0.003393962047994137, -0.6582104563713074, 0.35264432430267334, 0.1578475385904312, 3.149354934692383, -0.2086050659418106, 0.17486850917339325, -0.683473527431488, 1.3494952917099, 1.5935237407684326, -0.8494670391082764, -0.08620300143957138, 0.13609814643859863, 0.5280500650405884, 1.322624683380127, -0.9547944068908691, -0.8138295412063599, 0.8672853112220764, 1.6469889879226685, -1.0930161476135254, 0.6716851592063904, -0.013794687576591969, -0.7431960105895996, 1.7984205484390259, 1.3724498748779297, -0.3032735586166382, -0.18131189048290253, -0.850201427936554, 0.21190601587295532, 0.6430598497390747, 0.6000176072120667, 1.5033154487609863, 0.03934710472822189, 1.0542104244232178, 1.0921164751052856, -0.7221499681472778, -0.567391574382782, 0.9480128288269043, -0.5392935276031494, 1.603635311126709, -0.9666173458099365, -0.09941236674785614], [0.279122531414032, 2.1076347827911377, -0.07671777158975601, -0.9488210678100586, 0.9750963449478149, 0.27739420533180237, 0.7543531656265259, 0.6021396517753601, 1.3608512878417969, 1.1035090684890747, 2.581472635269165, -1.2286460399627686, 0.5225597620010376, -0.04482525587081909, 0.3695809245109558, 1.3012921810150146, 0.7668343186378479, -0.9569141268730164, 0.2053537517786026, 0.7605693936347961, -0.1551106572151184, -1.0598238706588745, -0.07044205069541931, -1.4628150463104248, 0.5905495285987854, 0.5170887112617493, 0.42514243721961975, -0.5758149027824402, 0.43910497426986694, -0.42437979578971863, 1.1030739545822144, -0.506361722946167, 1.7157371044158936, -0.5065728425979614, -0.913571298122406, -0.6940438151359558, 0.4287305772304535, -0.6474999189376831, -0.9461334347724915, -0.422249972820282, 0.026684381067752838, 1.7370429039001465, 0.3047393262386322, -0.5513588190078735, 0.4119945466518402, 1.6268634796142578, 1.4531874656677246, 0.34019935131073, -0.8735553026199341, -1.6781790256500244, -0.46546506881713867, 0.2711125314235687, 2.1396942138671875, -0.5471006035804749, 0.9000056385993958, -1.2852803468704224, 1.69222891330719, 0.8097756505012512, 0.43997201323509216, -0.6527765989303589, -1.1013226509094238, -0.24175341427326202, -0.03710946440696716, 0.542445719242096, -0.9146426916122437, 1.087125539779663, -1.2096387147903442, 0.305081307888031, 0.5044463276863098, -1.391957402229309, -0.13964004814624786, 0.0020135617814958096, -0.012013910338282585, -1.9707043170928955, -0.41489264369010925, 0.6207966804504395, -0.198699951171875, 0.30700644850730896, 1.1008622646331787, 0.5889461636543274, 0.6966391205787659, -0.8406205177307129, 0.5592449307441711, 0.39859771728515625, -0.35758236050605774, 1.270513892173767, -1.1013548374176025, -0.4437709450721741, 0.8036463856697083, 1.0880186557769775, 1.5890612602233887, -0.6549800038337708, 1.9942991733551025, 1.7117533683776855, -1.4536230564117432, -0.848849356174469, 1.6876682043075562, 0.5341471433639526, -0.07249964028596878, -1.4317547082901, -0.3639697730541229, 0.15095800161361694, 1.5569363832473755, 1.851758360862732, -1.058203935623169, -0.3957469165325165, -0.12108700722455978, -0.5285400152206421, -0.19818046689033508, 0.6866239905357361, 1.509106159210205, -0.12607495486736298, 0.6683463454246521, 0.4194137752056122, 0.600058913230896, -0.15280792117118835, 0.4022162854671478, 0.37371566891670227, -0.8022723197937012, 0.364270955324173, -0.08747054636478424, 0.30075517296791077, 0.41486117243766785, -0.37706732749938965, -1.747657299041748, 0.02164984866976738, 0.16478845477104187, -0.8981328010559082], [-0.13903303444385529, 0.06602021306753159, -1.167346715927124, 1.3095393180847168, 2.0106523036956787, 0.8927978277206421, -0.19109377264976501, 0.39663127064704895, 1.4656026363372803, -0.671330988407135, -2.4707634449005127, -0.40543729066848755, 1.4977163076400757, -0.8059950470924377, -1.0873500108718872, -0.8591304421424866, -0.8747830986976624, -0.23063445091247559, -0.7773882746696472, -0.5443016886711121, -0.146551251411438, 2.0194859504699707, -2.5505387783050537, -0.9225045442581177, 0.23117195069789886, -0.36799320578575134, 1.1977488994598389, 1.353004813194275, 0.3333771824836731, 0.5199328660964966, -2.532224416732788, 0.3177785277366638, -1.3931853771209717, -0.1524633765220642, 1.3334800004959106, -0.24388086795806885, -0.5841124057769775, -0.29559826850891113, -0.15841808915138245, -0.7154936194419861, 0.4873853325843811, 0.9702150225639343, 0.6392775774002075, -0.9478472471237183, -0.8470709919929504, 0.2512061595916748, 0.6690705418586731, 0.5198219418525696, -0.24113868176937103, 0.6648495197296143, -1.4347413778305054, -1.0354487895965576, -0.8806383609771729, -1.25295889377594, 0.2989976108074188, 0.30948498845100403, 1.3168983459472656, -0.4540809690952301, -0.28120025992393494, -1.6095774173736572, 1.2485462427139282, -0.24485211074352264, -0.730504035949707, -0.3822292983531952, 0.6385628581047058, 1.416020154953003, 0.6071968078613281, -2.230743646621704, -0.13752521574497223, -0.8721665143966675, -1.4271905422210693, -0.11338590085506439, -0.49693191051483154, 0.2618204355239868, -0.28070712089538574, -0.08639122545719147, -1.1694903373718262, 1.8890388011932373, -0.0936380922794342, 1.0224218368530273, 0.6173871159553528, 0.2782335579395294, 0.8845202326774597, 0.5137075781822205, -0.6753848791122437, 1.3342365026474, 0.8279340267181396, 1.4663114547729492, -0.8800851702690125, 0.857954740524292, -1.1457475423812866, -1.122507929801941, -0.12161347270011902, 0.544864296913147, 1.4300564527511597, -2.7789061069488525, 0.08386284112930298, 0.452572226524353, -1.41896390914917, -0.1899852305650711, 1.7786685228347778, 1.3011400699615479, -0.20437222719192505, -0.2631521224975586, 0.012074715457856655, 0.3233412206172943, -0.7508811950683594, 0.38254907727241516, 0.8251510262489319, -0.08475053310394287, 0.09967731684446335, 0.5647013783454895, 0.6673690676689148, 1.8052797317504883, 1.1524219512939453, 0.10174059867858887, 2.125392436981201, 0.9156227111816406, 0.248674213886261, 3.058110475540161, 2.090238094329834, -0.0167061910033226, 0.7289265990257263, -0.3421977758407593, -0.8466348648071289, -1.1794087886810303, -1.4867477416992188, -1.0856791734695435], [-1.2383160591125488, 1.568437099456787, -1.3192787170410156, -1.5135948657989502, 0.5193935632705688, 0.3035900592803955, 1.247172474861145, 0.13191282749176025, -1.2467727661132812, 0.5969904661178589, 0.09616678208112717, -0.1547747552394867, -0.7873863577842712, 1.3108460903167725, -2.429718255996704, -0.5370407104492188, 0.19780078530311584, 2.5860204696655273, -0.191285640001297, -0.26232683658599854, 0.5391270518302917, -0.7028310298919678, 1.4375399351119995, -0.6709633469581604, 0.4270976185798645, -1.0170689821243286, 0.20626628398895264, -1.125869631767273, -0.3081973195075989, -0.6716747879981995, -0.9853718280792236, 1.1536118984222412, 0.8556072115898132, -0.9571676254272461, -0.17875994741916656, 0.37613821029663086, 1.2592099905014038, 0.48592689633369446, 0.33810490369796753, -0.7603920102119446, 0.5948973298072815, -0.1969200074672699, 0.8621121048927307, -0.05828092619776726, -0.29177534580230713, 0.8266951441764832, 1.3827952146530151, 0.9308211803436279, 0.1227603480219841, -0.9528909921646118, 0.7539242506027222, -0.5104163885116577, 0.3677355945110321, 0.37924838066101074, -0.5478004813194275, -0.5895314812660217, -0.019157597795128822, 0.44681277871131897, 0.5579231381416321, -0.03825082257390022, -0.7113467454910278, 1.8529186248779297, -0.7180847525596619, 1.1783427000045776, 0.5865811109542847, 0.2913132607936859, -0.32320666313171387, -0.8328267931938171, 1.8690707683563232, 0.7976839542388916, 0.7407416701316833, -0.09560508280992508, 1.158666729927063, 0.682809591293335, 0.04987826570868492, -1.0819861888885498, -0.3806068003177643, 0.9034691452980042, -1.2559723854064941, 0.3566667139530182, 3.0634379386901855, 1.0757437944412231, 0.08184616267681122, 0.9663999676704407, 0.03842156007885933, -0.9032833576202393, -1.014224886894226, 0.7059679627418518, -1.1819382905960083, 0.6973797678947449, 1.2462198734283447, 0.10445693880319595, 1.842391014099121, 0.02156577631831169, 0.9937222003936768, -1.4815384149551392, -0.7385396361351013, -0.8867174983024597, -1.316084384918213, 0.7425537705421448, 0.664377748966217, 0.3564436435699463, 1.9166325330734253, 0.5342329740524292, -0.8699139952659607, 2.3924460411071777, 0.29271188378334045, 1.42911958694458, 1.1980020999908447, 0.2637402415275574, -0.3118532598018646, 0.9875304102897644, 0.9390414953231812, -0.030752897262573242, 1.5950406789779663, -0.24744315445423126, -0.5852847695350647, -0.38883480429649353, -0.7241402268409729, 1.1279386281967163, 0.7150726318359375, 1.196340560913086, -1.9544572830200195, -0.22563832998275757, 0.4113830029964447, 0.4535437822341919, -1.2767330408096313, 0.4403323829174042], [0.9924085736274719, 0.3154467046260834, 0.2795262634754181, 1.6672519445419312, -0.7568599581718445, 0.38853511214256287, -1.1328907012939453, -0.47157368063926697, -0.28539586067199707, 0.04221147671341896, 0.6942878365516663, 0.12291520833969116, -0.05227401480078697, -0.9225214719772339, -0.36043688654899597, 0.1883227378129959, 1.1057771444320679, 2.006748914718628, 0.5827542543411255, 1.2962825298309326, 0.35929766297340393, 1.1205774545669556, -0.37390241026878357, 0.022029398009181023, -0.41571044921875, -1.0199617147445679, -0.27446961402893066, 0.2679520547389984, 1.2888314723968506, -1.2668859958648682, 1.8328455686569214, -0.2662321627140045, 1.3057708740234375, 0.1687747985124588, 0.4462631046772003, -0.6042040586471558, -0.1785356104373932, -0.9378975033760071, 0.31100940704345703, -0.7799066305160522, 2.0065114498138428, -2.362295627593994, 0.06845235824584961, 0.6096667051315308, -1.521628737449646, 0.5486116409301758, 1.8495746850967407, -0.599219560623169, 0.4348025918006897, -1.003923773765564, 0.17610685527324677, -0.19514229893684387, -1.106677532196045, 0.12953335046768188, 0.5628364086151123, 1.0251399278640747, -1.1204825639724731, -1.2902960777282715, -1.829228162765503, -0.8917286396026611, -1.2253928184509277, -1.1313575506210327, -1.101321816444397, -1.072791576385498, 0.9819594025611877, -0.11333275586366653, 0.6644616723060608, -0.2945912182331085, 0.06042197719216347, -0.1430051028728485, -1.7159903049468994, -1.6099873781204224, -0.8683305382728577, -1.0572775602340698, 0.5957267880439758, 0.7568184733390808, -2.2345340251922607, -1.6908445358276367, -0.07766905426979065, -0.6313027143478394, -0.8992658257484436, 0.9767758250236511, -0.44258689880371094, 0.173512265086174, 1.7677104473114014, 0.26660192012786865, -0.11923612654209137, 0.3556959331035614, 0.30802395939826965, 0.6683254241943359, 0.1415046751499176, -0.3375481367111206, -0.1889275759458542, 0.8462281823158264, 0.4184674918651581, -1.03959059715271, -0.6012322902679443, -0.8792601227760315, 0.6348747611045837, 1.3217474222183228, -2.1789865493774414, -2.0722994804382324, -1.6452622413635254, -1.9813672304153442, 0.5195600986480713, 0.5721747279167175, 0.6957330703735352, -0.23283027112483978, -1.652726411819458, -0.08281341940164566, -0.6861754655838013, -0.9563804864883423, 0.5428809523582458, 0.23791208863258362, -0.9383929371833801, 0.3744959533214569, 0.1748623549938202, -0.16700617969036102, 1.778151512145996, -0.7305232882499695, 0.9431300759315491, -0.7361191511154175, -0.07395871728658676, 0.6755782961845398, -0.34259912371635437, -1.8321483135223389, 0.1455700546503067, 1.148250937461853], [-1.9274816513061523, -0.17618273198604584, 0.41245391964912415, 0.5024548768997192, 0.2490478903055191, -0.06714358180761337, -0.003385168267413974, 0.42663508653640747, -1.9893804788589478, 0.5751867890357971, -1.3406730890274048, 0.487829327583313, 0.5408231616020203, -0.9512893557548523, -0.030998097732663155, -0.49144628643989563, -1.0807219743728638, -1.427169919013977, 1.866140604019165, 1.8983235359191895, 0.4514476954936981, -0.652694046497345, -0.3579866290092468, -0.6852094531059265, 0.6068771481513977, -2.7930262088775635, -0.8386655449867249, 2.4982309341430664, 1.526297926902771, 0.03813338652253151, 0.5603317618370056, -1.5458780527114868, -0.3930949866771698, -0.07905204594135284, -0.47085678577423096, -0.035802584141492844, 0.012736459262669086, 0.5258583426475525, 1.2090332508087158, -1.634726881980896, 0.3237203359603882, -0.8765354752540588, 1.7616022825241089, -0.17308764159679413, 0.5012949705123901, 0.5812623500823975, 0.8436272144317627, 0.4656875431537628, 1.3158529996871948, -0.8353280425071716, -0.20864446461200714, -0.3214658796787262, -1.4328210353851318, 2.392960786819458, 1.363814115524292, -1.0600273609161377, 0.5896530747413635, 1.1401766538619995, -0.8419948220252991, 0.03550206497311592, 1.7093740701675415, 1.2753188610076904, -0.5298274755477905, 0.13744623959064484, 0.45025157928466797, -0.30678504705429077, 0.38547712564468384, 0.010622942820191383, 0.13775242865085602, -0.37347322702407837, -0.8147854208946228, -0.9551987648010254, -0.01906852424144745, -1.769195318222046, 0.9205381870269775, -1.1072958707809448, 0.9510531425476074, -0.4523358941078186, 1.3932912349700928, -0.20792341232299805, -0.24325917661190033, -0.6063350439071655, 1.6544103622436523, -1.926267385482788, -1.1387563943862915, -1.5400571823120117, -1.3149596452713013, -0.3203709125518799, 0.26825571060180664, -1.4256559610366821, -2.0130584239959717, -0.24960584938526154, 0.5572265982627869, -0.5497238636016846, 1.076372504234314, -0.062026530504226685, -0.8777298331260681, 0.7416831851005554, -0.5402707457542419, -0.4009617865085602, 0.3978212773799896, -1.3694086074829102, 0.147314190864563, -1.1776752471923828, -0.5852366089820862, -0.8597279191017151, 0.5971112847328186, -1.561689019203186, -0.0014509857865050435, -0.39505115151405334, -0.647907555103302, 1.530669093132019, 0.13664263486862183, 0.46128442883491516, -2.1374502182006836, 0.37661346793174744, 0.3557491600513458, 0.17849235236644745, -0.08805417269468307, 0.0954999178647995, 1.3297874927520752, -0.6382498145103455, 0.729119062423706, 1.7760969400405884, 1.8603847026824951, -1.502375841140747, -0.930615246295929, 1.0113009214401245], [-0.7691732048988342, 1.7016081809997559, 1.5476452112197876, -0.328575074672699, 1.6237610578536987, -2.9123945236206055, -2.4241225719451904, -0.9777380228042603, -0.36391836404800415, -0.8900195360183716, -0.3469884395599365, 0.44944724440574646, -2.706183671951294, -0.5161762237548828, -0.1955806165933609, -0.3711612820625305, 0.0049304901622235775, 1.913141131401062, 1.7575129270553589, 0.7744871377944946, -0.29306530952453613, -1.0919374227523804, 0.07355771213769913, -0.8726858496665955, -0.09946359694004059, 0.26126953959465027, 2.8298535346984863, 0.6714100241661072, -0.9087213277816772, 0.05022136867046356, -0.29825153946876526, 0.7513856291770935, 1.03369140625, -0.29941415786743164, -0.3349582552909851, -0.8120465278625488, 1.5006033182144165, 0.26509883999824524, -0.6397119164466858, 1.8078365325927734, -0.007823165506124496, -0.6740851998329163, -1.2863430976867676, -0.6167706847190857, -0.9015285968780518, -0.15089550614356995, -0.04508016258478165, -1.1818714141845703, 1.4915319681167603, 0.5162193775177002, -0.33919116854667664, -0.14365987479686737, 1.1223725080490112, -0.5523073077201843, 1.6196584701538086, -0.24200895428657532, -0.8184183835983276, 0.6335077881813049, 1.2236266136169434, -0.3005315661430359, 0.39337924122810364, 0.42118456959724426, 1.0234367847442627, 0.618061363697052, -1.6755220890045166, 0.9867550134658813, -0.422149121761322, -1.0164015293121338, 0.29559171199798584, -0.5675992965698242, -1.0797849893569946, 0.9178088903427124, -1.4126582145690918, 0.8015571236610413, 0.4839235246181488, 0.8053669333457947, -1.395574688911438, 0.11174960434436798, -0.7471916079521179, -0.17210634052753448, -0.6519132852554321, -0.33932262659072876, 0.331127405166626, 0.784430205821991, 0.9669327735900879, -1.1811342239379883, 1.0187175273895264, 1.4531456232070923, -1.0180597305297852, -0.5799934267997742, 1.9517810344696045, 0.17909595370292664, 1.485393762588501, 0.36560526490211487, -0.9917495250701904, -0.021988818421959877, -0.4396221339702606, -0.17902177572250366, 0.6032326817512512, -0.02508525177836418, 0.9986236095428467, 0.40559840202331543, 0.390198290348053, -0.32485464215278625, -0.08399451524019241, 0.984472393989563, 1.1647684574127197, 1.3043270111083984, -0.1685451716184616, -0.38723400235176086, -1.4575127363204956, 1.4966987371444702, -0.16976645588874817, 0.5324052572250366, 0.7903522849082947, 0.46152934432029724, 0.860732913017273, 0.9213306903839111, 0.35681259632110596, 0.7344946265220642, 1.7402223348617554, 1.7277647256851196, 0.43415138125419617, -0.059802837669849396, -0.4849281311035156, -1.4640592336654663, 0.15317584574222565, 0.19822484254837036], [0.8744598031044006, 0.14183150231838226, 1.2126266956329346, -1.2013978958129883, 0.06624893844127655, -0.08385401219129562, 0.5707110166549683, 0.24847708642482758, 0.37461981177330017, 1.6554524898529053, 1.8442726135253906, -1.1603816747665405, 0.5635634660720825, -1.4475375413894653, 1.491501808166504, -0.15400543808937073, 1.2857164144515991, -0.2792331278324127, -0.4454982280731201, 0.8187977075576782, -0.7200029492378235, 0.2979460656642914, -0.05385737866163254, -0.6097978949546814, 1.1515074968338013, 0.6389193534851074, -0.26086515188217163, 0.3005134165287018, -1.295318841934204, -1.3766800165176392, -1.0744655132293701, 1.2155770063400269, 0.4876847565174103, -0.4292871057987213, 2.085130453109741, 1.9855469465255737, 0.2855030596256256, -0.1542457789182663, -0.7307377457618713, -0.8877537846565247, 0.49472033977508545, -1.1767131090164185, 0.5801167488098145, -0.051782507449388504, -0.44178009033203125, 1.7740020751953125, 0.4920386075973511, 0.11350374668836594, -0.3345125615596771, -1.0897465944290161, -0.9980124235153198, -0.49252521991729736, -0.17112620174884796, 0.1362423151731491, -0.3836490511894226, 0.870789110660553, 0.7714040279388428, 0.763580322265625, -1.484119176864624, 1.4231234788894653, -1.2350071668624878, -0.39155372977256775, -0.6586464643478394, -3.436370611190796, -1.3384979963302612, -0.3292049467563629, -0.9886521100997925, -0.3787255585193634, 0.5309578776359558, -0.08665972203016281, -1.0317820310592651, -0.8659258484840393, -0.027606099843978882, 2.3110220432281494, 0.2804272770881653, -1.4700754880905151, -0.12520678341388702, -0.13977956771850586, 0.09203166514635086, 0.6447827219963074, -0.38986170291900635, 0.9869094491004944, 0.5147291421890259, -0.4156312942504883, 0.28200721740722656, 1.2933207750320435, -1.211423635482788, 0.18217796087265015, 0.9556325078010559, 0.444364070892334, 0.6407495141029358, -2.017402410507202, 2.435264825820923, 1.0376896858215332, 1.2963205575942993, 0.17640851438045502, -0.42138758301734924, -0.47894957661628723, 0.38420620560646057, -0.8802123069763184, 0.4671643376350403, 2.8473222255706787, -0.3799363672733307, 0.7766370177268982, 0.24969838559627533, -1.5111865997314453, 0.25291410088539124, -1.503721833229065, -2.262381076812744, 0.09280361235141754, 2.415539026260376, -0.024698909372091293, 0.4188784956932068, -0.5536078810691833, -0.6671620607376099, -0.07814500480890274, -0.8878389596939087, 1.824457049369812, -0.583995521068573, -0.8266393542289734, -0.9037846922874451, 1.5186777114868164, 1.7950005531311035, 1.097961664199829, -0.7377866506576538, 1.3059132099151611, -1.0207479000091553, 1.0334640741348267], [1.4374825954437256, -0.574453592300415, 0.8633168339729309, 0.7754034996032715, -0.660129189491272, -1.3101494312286377, -0.3696020841598511, 1.568922758102417, 1.6162039041519165, 0.09845668077468872, -0.48472657799720764, 0.03776749223470688, -0.9008498191833496, -1.3641983270645142, 0.75816810131073, -1.0457675457000732, 0.6776201725006104, -1.285532832145691, -1.1065118312835693, -0.5564354658126831, -0.2943045496940613, 1.246104121208191, 0.05724004656076431, -1.006136417388916, -1.6468734741210938, -0.4413450360298157, -0.5422475934028625, 1.2089484930038452, -0.15998922288417816, -0.9840403199195862, 0.8187069892883301, 0.2912885844707489, 1.1156525611877441, -1.105383038520813, -1.912556529045105, 1.5508615970611572, -1.7072105407714844, 1.3376573324203491, 1.3186243772506714, 2.1488394737243652, -0.2029801607131958, 1.5323196649551392, 1.2125580310821533, 1.1411782503128052, 1.1715542078018188, -0.06689909100532532, 1.6262716054916382, 1.2875370979309082, -0.1646469086408615, 0.24898582696914673, -1.3402323722839355, -0.6533209681510925, 1.7705668210983276, -0.9041174054145813, -1.9845913648605347, 0.26711326837539673, 1.0171433687210083, -0.6527009010314941, -1.0294874906539917, -1.4375871419906616, -0.3581988215446472, -1.1731598377227783, -0.3230791389942169, 0.45038530230522156, -0.6362849473953247, -0.7802286744117737, -0.7826669216156006, 0.23180079460144043, 0.25078800320625305, 0.06618111580610275, -0.7718246579170227, 0.3229278028011322, 2.0670900344848633, 1.0626405477523804, 0.7641012668609619, 2.9019851684570312, 0.611305832862854, -0.26749494671821594, 0.5684085488319397, -0.44021016359329224, -0.6648436784744263, 0.5298272967338562, 0.2932334840297699, -0.5067629814147949, 1.6322461366653442, -2.060947895050049, -0.4378184974193573, 0.7775707840919495, 0.5244945287704468, -0.9639666080474854, 0.2421625554561615, -1.0382598638534546, -0.804384171962738, 1.6531031131744385, 1.1725152730941772, 0.028038306161761284, 0.9901683926582336, 0.3865627348423004, 1.4780263900756836, 0.09130120277404785, -1.1348718404769897, 0.6504960656166077, -0.5807345509529114, 0.8410465717315674, -1.666364312171936, 0.13488908112049103, 1.2362703084945679, 0.8764377236366272, -0.5459675192832947, 0.5266079902648926, -0.9578190445899963, -0.632973849773407, -1.1608377695083618, -0.5396175384521484, -0.1352996826171875, -0.9501007795333862, 0.5691216588020325, -0.9353757500648499, 0.693694531917572, -0.38217881321907043, 1.1395092010498047, -0.32099053263664246, 0.01678820699453354, 0.9652231335639954, -0.277145117521286, -0.944614827632904, 0.6191005706787109, 0.37214866280555725], [1.0344271659851074, 0.33727285265922546, -0.690311849117279, -0.20373597741127014, 0.8255988955497742, 0.24279829859733582, -0.6409341096878052, -0.3003085255622864, -0.6049727201461792, -0.6859170198440552, -0.40260443091392517, 1.1502712965011597, -0.23661565780639648, -1.0288852453231812, -1.0310362577438354, 0.7473437190055847, -0.34481117129325867, 1.7823371887207031, -1.5614439249038696, -0.4299772083759308, -1.5612446069717407, 0.28392815589904785, 1.13560950756073, 0.8358489274978638, -0.017579803243279457, -1.1275535821914673, 1.4726454019546509, 0.06673812121152878, 0.1681557148694992, -1.8495121002197266, 1.9529026746749878, -0.49381351470947266, -1.815507411956787, -0.3243522644042969, 0.4128056764602661, 0.344144344329834, -0.2742269039154053, -0.4561605751514435, 0.6606107354164124, 0.9064153432846069, -0.648047685623169, -0.1580141931772232, -0.9732648730278015, -0.2924664318561554, -1.5370843410491943, 0.38830050826072693, -1.9611916542053223, -0.23519086837768555, -0.1759405881166458, -0.8385758996009827, -1.0244216918945312, -0.5289813280105591, -0.07049427181482315, -2.04303240776062, 1.183236002922058, 1.6937754154205322, 0.7919897437095642, 0.9430707097053528, 0.36046162247657776, 0.9317947626113892, 1.7372697591781616, 0.6928703784942627, -0.08918885141611099, 1.1349132061004639, 1.2121881246566772, 0.9886146187782288, -0.5219687223434448, -0.6585291624069214, -1.1552175283432007, 0.7433043718338013, -1.336444616317749, 0.7955812811851501, 1.2348179817199707, 0.20959196984767914, 0.0658283531665802, 0.9055044054985046, -1.1967321634292603, 0.42866024374961853, -0.7699609994888306, -1.4988733530044556, 1.3542474508285522, -0.047367196530103683, -1.302756905555725, 0.501127302646637, 0.5750049948692322, -0.6242141127586365, 1.10995614528656, 1.6700071096420288, -0.43032124638557434, 0.17862941324710846, -0.5518516302108765, -0.23736710846424103, 1.1414377689361572, 0.4009106159210205, -0.5641553997993469, -1.1122426986694336, 1.1780219078063965, -0.14958544075489044, 1.88251531124115, -0.6744577288627625, 0.3358865976333618, -0.3752298057079315, -2.073990821838379, -0.8997208476066589, -0.13629867136478424, -1.579176902770996, 0.021501967683434486, -1.5523570775985718, 0.7534142732620239, -0.24515396356582642, -2.0322508811950684, -0.09090140461921692, 0.5011346340179443, -0.41590604186058044, -1.3753931522369385, -1.0289287567138672, 1.3910242319107056, -0.039408378303050995, 2.3992767333984375, -0.010637094266712666, 1.2905670404434204, 1.5987352132797241, -0.508399248123169, -1.1628644466400146, -0.13457359373569489, 1.4050425291061401, -0.07156286388635635, 0.19809377193450928]], [[-2.2196600437164307, -1.3464007377624512, -0.6545947194099426, -0.959442675113678, 0.8528739809989929, -0.5034288763999939, 0.7958409786224365, 0.07455290108919144, -1.5303430557250977, -0.07043524831533432, -0.7438104152679443, 0.6511693000793457, -0.3592771589756012, -0.04798189923167229, -1.3196371793746948, -0.07925508916378021, -1.5785741806030273, -0.5987454652786255, -1.086182951927185, 0.06704819947481155, -0.2931286096572876, 0.6817260980606079, -0.22077502310276031, 0.20012745261192322, 0.29536691308021545, -0.19247233867645264, -0.7127120494842529, -0.5545034408569336, 0.037123240530490875, 1.2108829021453857, 0.015472391620278358, -0.9963089227676392, 1.0729578733444214, -0.33324185013771057, -1.0013433694839478, -0.7105982899665833, -0.9266968369483948, -1.2578762769699097, -0.8951536417007446, -1.0388799905776978, 1.4811875820159912, 0.4164853096008301, -2.643925189971924, -0.682742714881897, -2.1528124809265137, 0.34242498874664307, 0.08705703914165497, -0.10523273050785065, 0.8323389291763306, 3.0229110717773438, -0.12272439897060394, -0.25665029883384705, -0.13854601979255676, -1.8519749641418457, 0.23441669344902039, 1.5189703702926636, 1.0087604522705078, -1.6505053043365479, -1.305185317993164, 0.9452855587005615, 0.4398966133594513, -0.44967368245124817, 0.8748542666435242, -0.2963629961013794, -0.01658182218670845, 0.890783965587616, 0.1600342094898224, 0.0904947891831398, 2.091223955154419, 0.813826858997345, 1.0627896785736084, 0.5748802423477173, 0.5393173098564148, -0.4126529097557068, -1.3011420965194702, 3.3601231575012207, 0.058753326535224915, -0.31994372606277466, -0.44403076171875, -1.1591442823410034, 0.13085201382637024, 0.6013911366462708, 1.3480031490325928, 0.03557504341006279, -0.4409061372280121, -1.0489184856414795, -0.5526655316352844, -0.9804506897926331, -0.5396826863288879, 0.9627377390861511, 0.8540388941764832, 0.5869515538215637, -0.5448237061500549, -0.07127130031585693, -1.2129464149475098, -0.7774571776390076, 0.07376501709222794, -0.8481366634368896, -1.3716425895690918, 0.3394908308982849, 1.0052862167358398, -0.057965975254774094, 0.6883795857429504, 0.24324066936969757, 0.2835044264793396, -0.13026125729084015, -2.257004737854004, -1.383558750152588, 0.9615219831466675, -2.392183542251587, 2.1911203861236572, 0.013767720200121403, 0.1522187441587448, -0.0019582631066441536, 0.1339920610189438, -0.8179267644882202, 0.4180845618247986, -0.008982229046523571, -0.5210729837417603, -0.4081745743751526, -1.3483538627624512, 1.7374802827835083, -0.5809486508369446, -0.10756398737430573, 0.1715036779642105, 1.4821542501449585, -0.3021247088909149, 0.4951399564743042], [-0.9036834836006165, -1.2583487033843994, -0.14599068462848663, -0.7148529887199402, -0.7384032011032104, -1.1636089086532593, -1.4571950435638428, 0.4111836850643158, 1.2708885669708252, -0.46989357471466064, -0.8935421705245972, 0.7272721529006958, 0.48816144466400146, -0.486387699842453, -0.5908181667327881, 0.08293195068836212, 0.4872058033943176, 1.6094926595687866, -0.03003111109137535, 0.7195187211036682, 1.8501157760620117, -1.153403878211975, 0.1591515988111496, 0.41553133726119995, 0.856853723526001, -0.08689059317111969, -1.8574141263961792, -1.1590641736984253, -1.1929446458816528, 0.8714562058448792, -0.39829614758491516, -0.13376261293888092, -0.8598914742469788, 0.7794362306594849, -0.18794123828411102, -0.2592126429080963, -1.2194594144821167, -1.781198501586914, 1.542944073677063, 1.656419277191162, -0.683641791343689, -1.4540274143218994, -1.3391951322555542, -0.6978335380554199, 0.23621517419815063, 0.6657209992408752, -0.0511711947619915, 2.0042030811309814, -1.7802612781524658, -1.2466760873794556, 1.8209567070007324, -0.8588348627090454, -0.06020275875926018, 0.5966549515724182, -1.9626691341400146, -0.003367338329553604, 1.0082099437713623, -0.14282925426959991, -0.8150356411933899, 0.47148481011390686, 0.4575744569301605, -0.5937486886978149, -0.6160427927970886, 0.04785636067390442, 0.44184499979019165, 0.4776690900325775, -0.41863730549812317, -2.0832607746124268, 0.06010865792632103, 0.6625107526779175, 0.9857164621353149, -1.7382248640060425, -1.311630368232727, 1.4235129356384277, 0.9941041469573975, 0.8321911096572876, 0.22281301021575928, 2.213230848312378, 1.1469340324401855, 0.3036508858203888, 0.45621106028556824, -0.11948192864656448, 0.26120075583457947, -0.48379775881767273, -0.26642656326293945, -1.5975637435913086, -0.8271188735961914, 1.2236979007720947, -0.38187074661254883, -1.0506888628005981, 1.4242894649505615, 0.5426227450370789, -0.8568373918533325, 0.4482456147670746, 0.19715452194213867, 1.5071197748184204, 0.3470945656299591, 0.1970960795879364, 0.12405343353748322, -0.619274914264679, 0.4755820631980896, 0.7335637807846069, -0.3298220932483673, -0.9084072709083557, -0.019386155530810356, 0.472528874874115, -0.4254024922847748, 1.5234103202819824, -1.0402129888534546, 2.57552170753479, -0.18723812699317932, 0.21096856892108917, 0.02281181327998638, 1.7225950956344604, 0.8394994139671326, -1.5594977140426636, 1.8642855882644653, -0.8177210092544556, 1.1036734580993652, 1.7806023359298706, 0.08333379775285721, -0.32563483715057373, -0.8801791667938232, -1.7510101795196533, -1.1864718198776245, 1.1802529096603394, -1.5699512958526611, 0.8285934329032898], [-2.0994458198547363, 0.05421855300664902, -0.7494936585426331, -1.8225315809249878, 0.8496752381324768, -0.5499877333641052, -0.1845858097076416, 1.1196352243423462, -1.6942442655563354, 0.8981525301933289, 0.4771636128425598, 1.3721275329589844, 1.7112013101577759, 0.10674823820590973, 0.5717891454696655, -0.28950196504592896, 1.7433007955551147, 0.5898331999778748, 1.1542718410491943, -0.13269738852977753, -0.8502692580223083, 0.7367857694625854, 1.2996166944503784, 0.8401113152503967, 1.450703740119934, 0.17448046803474426, -0.7718396782875061, 0.3092024028301239, -0.17085057497024536, -0.7668923735618591, -2.060105085372925, 1.455325961112976, -0.5734541416168213, -0.8188499212265015, 1.445327639579773, -0.7037619948387146, -1.4940508604049683, 0.46892985701560974, 0.3282124698162079, 1.6004209518432617, -1.3690185546875, 0.4503359794616699, -1.9731348752975464, -0.4603251814842224, -0.7518000602722168, -0.6112489700317383, 0.6900187134742737, 1.0272711515426636, 1.251371145248413, -1.434931993484497, -0.32109224796295166, -0.3124277889728546, -1.248295783996582, 0.8308643698692322, 1.531062126159668, -0.21368184685707092, 1.174574851989746, 0.4705405831336975, -0.8759180307388306, 1.5606513023376465, 0.7810927033424377, -0.45445191860198975, 0.2704259157180786, -0.6671712398529053, 0.6239879727363586, 1.8782048225402832, -2.0360357761383057, -0.4156228005886078, 0.5842874050140381, -0.005649620667099953, -0.4660417437553406, 0.13375476002693176, 0.25546881556510925, 0.9676254987716675, -0.8510252833366394, -0.948365330696106, -1.6185044050216675, -0.8305209875106812, 0.7439432740211487, -0.268303245306015, -1.7220823764801025, 0.41284769773483276, -0.3217957019805908, 0.16177868843078613, 0.8414262533187866, -0.15340018272399902, 0.14043495059013367, 0.38878437876701355, 0.02003544755280018, -1.3835113048553467, 0.20051409304141998, 1.079817771911621, -0.45120400190353394, 0.8190128803253174, -0.7093933820724487, -0.37279805541038513, 1.0717077255249023, 0.39518117904663086, -1.09023916721344, 0.7267274260520935, -1.9234298467636108, 0.8569189310073853, -0.7396474480628967, -1.7635138034820557, 0.6056193709373474, -1.0273627042770386, 0.9843573570251465, 0.8626412749290466, -1.2009614706039429, -1.2140591144561768, -0.5761037468910217, -0.0947841927409172, 0.062471263110637665, 1.3924260139465332, 1.0491011142730713, -1.195710301399231, 0.6015118360519409, -0.2895680367946625, -0.37846052646636963, 0.30008843541145325, 1.0453197956085205, -0.5238250494003296, 1.1340856552124023, -0.3096535801887512, -2.3038330078125, 0.12953686714172363, -0.7617894411087036, 0.3075867295265198], [-1.7067782878875732, -0.8875566720962524, -0.2970784902572632, 0.36319565773010254, 0.6894354820251465, 0.6935310363769531, 0.6592443585395813, -0.9772756099700928, -1.8012615442276, 0.35048672556877136, 0.7464311122894287, -0.2711886167526245, -0.3079864978790283, -0.28435105085372925, -0.1576562225818634, -0.3009147346019745, 0.029913831502199173, 2.596012830734253, 0.8135695457458496, -2.119436502456665, -0.8412286639213562, 1.06785249710083, 0.07272927463054657, -1.2008494138717651, -0.30885350704193115, -0.8652153611183167, -0.982642412185669, -0.22514104843139648, -0.02330603636801243, 1.0524266958236694, -2.3372952938079834, 0.9563779830932617, 0.16185596585273743, -0.8586873412132263, -1.513811469078064, 1.0076313018798828, 0.9623023271560669, 1.3164873123168945, -0.34266576170921326, -0.886026918888092, 2.023563861846924, -2.727396249771118, 0.5135447382926941, 1.5152772665023804, 1.1809232234954834, -1.8462750911712646, 0.4397154748439789, -1.4608790874481201, -0.8281521797180176, -0.764086902141571, -0.45979639887809753, 1.4790056943893433, -1.2177860736846924, -0.33492526412010193, 0.06438276916742325, -0.8861730694770813, 0.3391934931278229, 1.1973330974578857, -0.4665601849555969, 1.3865838050842285, 0.2489175945520401, 0.9781849384307861, 0.1620721071958542, 0.28732332587242126, -0.4357169270515442, -0.7207181453704834, -0.10998246818780899, -1.1544910669326782, -2.118931770324707, 1.5061641931533813, 0.8799144625663757, -0.2956085503101349, 0.6024267673492432, 0.3155096173286438, 0.3605557680130005, -1.7882837057113647, -1.0334155559539795, -0.999902069568634, 0.9566937685012817, 0.7705487012863159, -0.940813422203064, 0.5775818824768066, 1.257617712020874, 0.181588813662529, -0.4668331742286682, -0.2706342935562134, 1.1941534280776978, 2.3958961963653564, -0.7263015508651733, 0.3682434558868408, -0.7555434703826904, 0.16771140694618225, 1.469896674156189, 0.10043520480394363, -0.25563547015190125, -0.9721359014511108, -0.12905552983283997, 0.25879815220832825, -0.610216498374939, 0.7447853684425354, -0.6402080655097961, -0.3065250813961029, 1.4715715646743774, -0.17181695997714996, 0.579037606716156, 0.6427819132804871, 0.6964656710624695, 0.9882299900054932, 0.5522328019142151, 0.9860027432441711, -1.9891058206558228, 0.16351836919784546, 0.3479580581188202, 0.7810726165771484, 0.013719670474529266, 0.8412578105926514, 1.0978621244430542, -0.43021467328071594, 0.10893166065216064, 0.8750125169754028, -0.7739697694778442, 0.20799461007118225, 0.9705243110656738, 0.4946201741695404, 0.7448326349258423, 0.08469929546117783, 0.7437859773635864, 1.1714762449264526], [0.27078983187675476, 0.1075824722647667, -0.6107252836227417, 0.4120122492313385, -1.2876570224761963, -0.38367390632629395, -1.0077953338623047, -0.20446008443832397, 2.1416194438934326, 0.10744655877351761, 0.5070770978927612, -0.4221441447734833, -0.090853251516819, 0.5943441987037659, 1.521491289138794, -2.457329750061035, -1.2648006677627563, -0.3457081913948059, 1.2791131734848022, 0.575832724571228, 0.11705361306667328, 0.5970700979232788, -0.05210147798061371, -2.4244606494903564, 0.6006945371627808, -0.4321378469467163, 1.6715351343154907, 0.19957508146762848, 0.013374141417443752, -0.6011403799057007, 0.19399544596672058, 1.0474722385406494, -0.20266525447368622, -0.5075523257255554, 0.8337116837501526, 0.5631906390190125, -0.2914477288722992, -0.23496972024440765, -1.2858262062072754, -0.14880461990833282, -0.9822036027908325, -0.7794473767280579, 0.08090586215257645, -0.8964560031890869, 0.36275070905685425, 1.3471978902816772, -0.8500112891197205, -1.1315531730651855, -2.2203891277313232, -0.1613488793373108, 1.5175772905349731, 0.7561984062194824, 2.550023078918457, -1.1266430616378784, 0.33846935629844666, 0.046843335032463074, 0.45854607224464417, -1.2205839157104492, -0.5600444078445435, -0.37976139783859253, 0.10629979521036148, 0.28541842103004456, -1.238335371017456, 1.614671230316162, -0.6713458895683289, -0.563165545463562, 1.001733422279358, 0.32843440771102905, -0.18673069775104523, -0.11092647165060043, -1.673032283782959, 3.0488297939300537, -0.20681777596473694, -1.5597494840621948, 0.6775209903717041, -1.4833699464797974, -0.49715089797973633, 1.5405913591384888, 1.897180438041687, -0.8270999193191528, 0.18828807771205902, 0.6492977738380432, 0.5120425820350647, 0.2066996693611145, 0.04960289224982262, -0.12564320862293243, -0.021930793300271034, 0.8657986521720886, 0.6193481683731079, -0.9867129325866699, -0.22746732831001282, 0.08320770412683487, 1.7259480953216553, -0.8315132260322571, -1.576488971710205, -0.8767779469490051, 0.2816445827484131, 1.8258740901947021, -0.1099986881017685, 0.3909032940864563, -1.060925841331482, -2.7279980182647705, 0.7159301042556763, 1.0273609161376953, 0.7354795336723328, -0.6966235041618347, 0.2037213146686554, -0.14139322936534882, -1.2839882373809814, -1.1988238096237183, -1.766493797302246, -1.7631734609603882, -1.9313733577728271, 0.01852208375930786, 0.5126844048500061, -1.2769591808319092, 0.10425659269094467, 0.91004478931427, -0.958751380443573, 0.4300117790699005, 1.4015966653823853, -1.165442943572998, 1.2452642917633057, -0.698196291923523, -0.9959759712219238, 0.644038200378418, -0.05923943221569061, 0.16692805290222168], [-0.2550850212574005, -1.402968168258667, 1.3940260410308838, 0.9339654445648193, -1.1114643812179565, 0.05462394282221794, -0.5124555230140686, 1.6924139261245728, -0.05848102271556854, 1.806640625, -0.026774263009428978, -0.4987839162349701, -1.481536626815796, -0.2090328484773636, -1.1113144159317017, 0.6536616086959839, 0.6003590822219849, 2.031897783279419, -1.2833257913589478, 0.8436367511749268, -0.9060117602348328, 0.9941948652267456, -2.134751319885254, 0.5044616460800171, -0.4570629596710205, 0.5162783265113831, 1.621176838874817, 0.38004302978515625, -0.42456763982772827, -1.2872296571731567, -0.6608202457427979, -0.9219319224357605, 0.5323014855384827, 0.5271620750427246, 0.08609438687562943, 0.5689190626144409, 2.3880093097686768, -1.3584381341934204, -1.3151859045028687, 1.1381213665008545, -1.043473243713379, -0.7293477654457092, -0.5071602463722229, -0.035419125109910965, -0.5452229976654053, 1.252477765083313, 0.1647779494524002, -0.8140556216239929, -0.8586055040359497, -1.4154894351959229, -0.2415049821138382, -0.46367955207824707, 0.8116708993911743, -1.8038673400878906, -1.413567066192627, -0.0547015517950058, -0.06829287111759186, -0.8356659412384033, 0.7902257442474365, -1.2010798454284668, -0.6284632086753845, 0.30575868487358093, 0.22239144146442413, 1.3168658018112183, -0.3941302001476288, 0.930332601070404, 1.3341245651245117, -0.7773066163063049, -1.0708280801773071, -1.084135890007019, 0.02458006516098976, -1.4752006530761719, 1.1526892185211182, 0.28200340270996094, -0.8630307912826538, -0.5093904137611389, 0.7540013194084167, -0.7201191782951355, 0.9571518301963806, -0.9142804741859436, -1.570540189743042, 0.035558030009269714, -1.2989250421524048, 1.4908738136291504, -0.5621728301048279, 0.09139943867921829, 1.2803523540496826, 0.25140708684921265, 0.8683701753616333, 0.2667592465877533, -0.516889214515686, -0.44758179783821106, -1.7627298831939697, 0.2451961636543274, 0.6750516295433044, -0.1609824001789093, 0.515378475189209, 0.019448887556791306, -0.09309874475002289, -1.8031857013702393, 0.5234379768371582, -0.4755622446537018, 0.10076596587896347, 0.7864382266998291, 0.4746784269809723, 0.912147045135498, 0.12915299832820892, 1.5198297500610352, -0.6120986342430115, -0.9673492312431335, -0.5361884236335754, 0.03778540715575218, 0.24761039018630981, 0.22872237861156464, 2.835733652114868, 0.20338214933872223, -1.0034828186035156, -1.8443708419799805, 2.4971625804901123, 0.6276214718818665, 0.6616392731666565, 0.8022154569625854, -0.6985502243041992, 1.046450138092041, -0.4111320376396179, 1.6059013605117798, -1.0478163957595825, -0.7142481803894043], [-0.27018824219703674, 1.7984877824783325, 0.11346229165792465, 0.033244118094444275, 1.3760263919830322, 0.8094627261161804, -1.0193209648132324, 0.04696930944919586, 3.2239997386932373, -0.09798850864171982, 0.25294795632362366, 1.0813225507736206, 1.8443756103515625, 1.2182151079177856, -0.3502940237522125, -0.1459643393754959, 0.7789705991744995, 0.86246258020401, -0.586299479007721, -0.22626785933971405, 0.897115170955658, -0.4728030264377594, -0.3492361903190613, -0.9866346120834351, 1.0570980310440063, 0.6809405088424683, 0.2409474402666092, -1.09026300907135, -1.1997051239013672, 0.04376320168375969, 1.1833330392837524, 0.760342001914978, -0.8270986676216125, 0.5558969974517822, 0.9507898688316345, -0.44126105308532715, 0.37513574957847595, -0.3282749652862549, 0.31939634680747986, 0.34508979320526123, -0.03341596946120262, 0.8009679317474365, -0.22259166836738586, 0.3710745573043823, 1.9716479778289795, 1.7377039194107056, 1.3582451343536377, -0.08945699036121368, -0.43031060695648193, -0.9991958737373352, 0.07485827058553696, 1.20132577419281, 1.2326104640960693, 0.9469018578529358, 2.0472795963287354, -0.5223474502563477, -0.7231286764144897, -0.6327270865440369, -1.2329635620117188, 1.4063800573349, -0.06538427621126175, -0.4903048872947693, 1.0275148153305054, -0.36972057819366455, -1.405168056488037, 1.1400066614151, 0.16430048644542694, 0.4517981708049774, 1.3397884368896484, 1.3908874988555908, -0.275117963552475, 0.6348562836647034, -0.11316902190446854, -0.12678895890712738, 0.4391557276248932, 1.0906352996826172, 0.5216554999351501, 0.7568861246109009, 0.28219619393348694, 0.8332005143165588, -0.22359178960323334, 0.29136034846305847, 0.15522216260433197, 2.948472738265991, 0.8020615577697754, 1.021866798400879, 1.4331861734390259, -0.5254683494567871, 0.40515515208244324, 0.4989670515060425, -0.004841636400669813, -1.2571427822113037, 0.6183796525001526, 0.04496657848358154, -0.012539861723780632, 0.09512238204479218, -0.7075293064117432, 0.7751535177230835, -1.6599780321121216, 0.2076730877161026, -0.1345369815826416, -0.09214671701192856, -1.3208472728729248, -0.5549353361129761, -1.5262552499771118, 0.07817372679710388, -3.4954402446746826, -0.09387627243995667, -0.4814658463001251, -0.9130909442901611, -1.0459849834442139, -0.6106440424919128, 0.032390180975198746, -0.7186418175697327, 0.21430370211601257, 0.19417008757591248, -2.3908519744873047, 0.0799073651432991, 0.9306930899620056, 1.7817625999450684, 0.39758867025375366, 0.707966685295105, -0.11702940613031387, 0.7312374114990234, -0.5241560339927673, -0.07360734790563583, -0.1278287023305893, -0.539142906665802], [-0.02848559059202671, -1.283606767654419, 0.47561636567115784, -0.3801243007183075, 0.4616818130016327, 1.6808606386184692, -1.1961419582366943, -2.6478962898254395, -1.2250670194625854, -0.3080276548862457, 3.137474536895752, -1.3430149555206299, -2.4640369415283203, -0.3912692666053772, 0.20652058720588684, -1.9166960716247559, 1.5617852210998535, -0.5487859845161438, -0.8140372633934021, 1.3001161813735962, -0.8829145431518555, -1.067934513092041, -0.6575241088867188, -1.1159495115280151, -1.3874295949935913, -0.32446518540382385, -0.006966510787606239, 0.21563269197940826, 1.7531801462173462, -0.5354973077774048, -1.2659982442855835, -1.454265832901001, 0.1011800542473793, -0.18469645082950592, -0.2550881505012512, 1.7230186462402344, 0.3617899715900421, -0.20820659399032593, -0.06937842816114426, 1.457334041595459, -0.8693588376045227, -0.5881571173667908, -0.9780504107475281, -0.5363770127296448, -1.98301100730896, -1.771828055381775, 0.03772134706377983, -0.16127189993858337, -0.2680274546146393, 1.9168888330459595, 0.23419898748397827, 0.5012416243553162, 0.07742871344089508, -1.8060287237167358, 1.6236581802368164, -2.340472459793091, 1.7086637020111084, 0.5712370872497559, 2.485635280609131, -0.7011800408363342, -1.1232988834381104, -1.3455283641815186, 0.1645660251379013, -0.15616382658481598, -1.9317502975463867, -1.5880777835845947, -1.1964668035507202, 2.5643067359924316, -0.09744767099618912, 1.4898858070373535, 0.857293963432312, 1.3711146116256714, 0.5116580128669739, 0.6261959671974182, 0.7722182273864746, 1.3327513933181763, 0.5239754915237427, -0.05156204104423523, 0.5118716359138489, 0.2768223285675049, 1.7346891164779663, -1.1320699453353882, -0.3820708096027374, -0.20837271213531494, 0.7312402725219727, -1.1765453815460205, 0.09820083528757095, 0.2934941053390503, 0.4410080015659332, -0.14642783999443054, 0.8901351094245911, -0.35871636867523193, -1.0570045709609985, 0.5990545749664307, 0.3572576642036438, 1.0126558542251587, 0.1729440987110138, -0.17544680833816528, -0.15547044575214386, -0.8455817103385925, 0.645858883857727, -1.143533706665039, 0.18097543716430664, 0.4337410628795624, -1.281639575958252, -0.18437057733535767, -1.145586609840393, -1.2570064067840576, -0.9540559649467468, -0.8204019665718079, -0.10018365830183029, -1.1784027814865112, -1.3490116596221924, -0.7441816329956055, -0.8392577767372131, 1.4295856952667236, -2.4810333251953125, -0.8510013818740845, 1.329606056213379, -2.0154497623443604, -0.8417921662330627, 2.361314296722412, 0.44465968012809753, -2.114219903945923, 0.6927576661109924, -2.5293259620666504, 0.027080334722995758, -0.9256666898727417], [-1.0145212411880493, 0.06567583233118057, -1.764407992362976, -0.9451361894607544, 0.6656559109687805, -0.4248507618904114, -0.7846670746803284, 1.7381436824798584, -0.3016063868999481, 1.0678346157073975, -0.6501278281211853, -0.9467922449111938, 0.15833702683448792, -0.49764829874038696, -0.7415640950202942, 1.832688570022583, 0.9975652694702148, -0.5330215692520142, 0.9926212430000305, 0.3450097143650055, -2.190159559249878, -1.1192858219146729, 0.4134390652179718, 1.2208149433135986, 0.2527925372123718, -0.17501363158226013, -0.29104146361351013, 0.02355526015162468, -1.5002561807632446, -0.3834519386291504, -0.24854956567287445, 0.8209061026573181, -1.0695995092391968, 1.287840485572815, 1.5295051336288452, 0.7680138945579529, -1.8644657135009766, 1.3794618844985962, -0.7404347658157349, -0.20674386620521545, -0.476961225271225, 0.5456463098526001, -0.6622015833854675, -1.3549751043319702, 0.7478327751159668, -1.893314242362976, 1.414713740348816, -0.7726339101791382, 1.1186894178390503, 0.5318055748939514, 0.17261871695518494, 1.051914930343628, 0.08618389815092087, -0.5724254250526428, -0.12073881179094315, 0.8157456517219543, -1.7569068670272827, 0.11877737939357758, 0.20230671763420105, -0.8098766803741455, 0.8075235486030579, 0.41146454215049744, 0.31497740745544434, -0.64427649974823, 0.9372661113739014, -0.5673393607139587, 1.6933420896530151, -1.1295744180679321, 1.1514942646026611, 0.6056162714958191, -0.6044449210166931, -0.1845542937517166, 0.833652138710022, -0.11261186003684998, -0.5780101418495178, 0.321098655462265, 1.4656518697738647, -1.4360548257827759, -0.3393973410129547, -0.41096457839012146, 0.6992623805999756, 1.3644487857818604, -0.9131103157997131, 0.5227442383766174, 0.7622100710868835, -1.1413532495498657, -1.7037913799285889, 0.946560800075531, -0.5948725342750549, 0.3697458803653717, 0.33996331691741943, 1.2912896871566772, -0.3571337163448334, 1.180791974067688, 1.1654167175292969, 1.946302056312561, -0.1932554394006729, 1.3629353046417236, -0.007866710424423218, 0.9264146089553833, -0.27375301718711853, 1.247009515762329, -0.8001329898834229, 0.6827213764190674, 0.7980422377586365, 1.0677069425582886, 0.8614432215690613, 0.409479558467865, 1.6554783582687378, -0.2608407139778137, 0.007626499980688095, -1.784169316291809, 0.2562713027000427, 0.5979797840118408, -1.3611223697662354, -1.1072391271591187, -2.8066565990448, 0.558746874332428, 0.05999220162630081, 1.9300998449325562, 0.5327141284942627, -0.20980346202850342, 2.789750814437866, -0.037852171808481216, 0.9671624898910522, 1.2319531440734863, 0.27946051955223083, 0.2908802032470703], [1.900665283203125, -1.548905372619629, 1.2724878787994385, 1.1578134298324585, 0.2609453797340393, -1.1989039182662964, -1.6278579235076904, 2.070679187774658, -0.7218493819236755, 0.85993891954422, -1.0655945539474487, -0.18568329513072968, -1.0406010150909424, -0.42476314306259155, 0.4154769480228424, -0.31995338201522827, 0.4810043275356293, 0.8256292343139648, -1.644300937652588, 1.0038173198699951, 0.0925111249089241, 0.3338342607021332, -0.8217787742614746, 0.5655045509338379, -0.38652661442756653, 0.31920281052589417, -0.14511071145534515, -0.08384771645069122, 0.10823670774698257, 0.670909583568573, -0.7384783625602722, 0.5019673705101013, -0.10166995972394943, -0.26797252893447876, -0.9741054773330688, -0.6683739423751831, 1.9032578468322754, -0.5758640766143799, -0.5713655948638916, -1.1231975555419922, 0.8024492263793945, -0.3035558760166168, 0.9127227067947388, 0.7855374813079834, -1.8627163171768188, 1.8804060220718384, -0.771412193775177, -0.7329438328742981, 0.2997995913028717, -0.7090579867362976, -0.5706571936607361, -0.9617298245429993, -1.0011136531829834, 0.9500221014022827, -0.5841365456581116, 0.2944156527519226, -0.6538920998573303, 0.06960860639810562, -0.6063957810401917, -0.7441114187240601, 0.4813040792942047, 1.0462377071380615, -1.3174165487289429, 0.9096676707267761, 0.0008936012163758278, 0.4797937273979187, 0.2881912589073181, -1.4769386053085327, 0.3888286352157593, 0.5353546738624573, 0.49695372581481934, 0.6951083540916443, -0.35184505581855774, 0.1883656084537506, 0.20878732204437256, 0.4067421853542328, 1.156895637512207, -1.0164979696273804, 1.9412378072738647, 0.6869062185287476, 0.9622466564178467, -0.34245964884757996, -0.5756792426109314, -1.066658854484558, -0.0323946550488472, 0.16518251597881317, 0.013689301908016205, -0.3040390610694885, 0.6621749401092529, -1.0331677198410034, 1.121293544769287, -1.2184187173843384, 0.07333924621343613, -0.49060410261154175, -1.562011957168579, -0.4728454053401947, -0.13513967394828796, 0.9583438634872437, -0.2303369790315628, 1.251552700996399, 0.9333856701850891, 1.542272925376892, 1.2084864377975464, -1.3967328071594238, 0.7260504364967346, -0.08754625171422958, -0.23351633548736572, -0.7533081769943237, -0.553867757320404, -0.46203669905662537, -0.09546098858118057, 0.5480346083641052, -0.42337846755981445, 0.36714446544647217, 0.5964148044586182, -0.11905097961425781, 0.47356462478637695, 0.9441995024681091, 0.5724157691001892, -0.19228583574295044, 0.0967661589384079, -0.02742111124098301, -2.848541736602783, -0.333787739276886, -0.03304649144411087, 0.0888482853770256, 0.05183377116918564, 1.1415066719055176], [-1.0075260400772095, 0.11853627115488052, 0.3291098475456238, -1.263472318649292, 0.02622750960290432, -0.09738040715456009, 0.9390558004379272, -1.5244479179382324, 0.7585954070091248, 0.12310250103473663, 0.1759396195411682, 0.7520075440406799, 0.4008546471595764, -1.8027551174163818, -1.8032244443893433, -1.9914144277572632, -0.4693301022052765, -1.5902413129806519, 0.3361207842826843, -0.11765773594379425, -0.2806428074836731, 0.6007952094078064, -1.2894576787948608, -0.08880940079689026, 1.1354622840881348, 1.431052565574646, 0.5062008500099182, -1.0080679655075073, -0.36466991901397705, -0.9044856429100037, 0.4912886321544647, -0.05325096473097801, -0.530570387840271, -0.097394660115242, -2.283747911453247, 1.5113884210586548, 0.1639435589313507, -0.9998903870582581, -1.3077218532562256, -0.8851490616798401, 1.7990758419036865, -0.5302613377571106, 0.3046240508556366, -0.9790553450584412, 1.7130706310272217, 1.0683693885803223, -0.7883510589599609, 1.7620892524719238, 1.2770808935165405, 0.28122884035110474, -1.7997578382492065, 0.23469336330890656, -2.290583848953247, 1.0804874897003174, 0.8977952599525452, -0.2171640247106552, 1.4861176013946533, 0.1859772503376007, -1.3925504684448242, -0.35831204056739807, -0.21839489042758942, 1.7005316019058228, -0.040482018142938614, -1.3815656900405884, 1.0518256425857544, 2.0229501724243164, -0.40795692801475525, 0.22101259231567383, -1.1766482591629028, -0.39864879846572876, 0.97202467918396, -0.10249101370573044, -2.213001251220703, -1.2982012033462524, 0.6940838694572449, -1.262450098991394, -0.6817672848701477, 0.13080739974975586, 1.0116599798202515, -0.6185317039489746, -0.8066197037696838, -0.09277927130460739, -0.8151479959487915, 0.009637989103794098, -0.03748730570077896, -0.15127059817314148, 0.31208837032318115, -0.004561166279017925, 0.6470738649368286, -0.817141592502594, 0.38170379400253296, -0.044813912361860275, 0.7181401252746582, -0.026812344789505005, 1.143833041191101, 0.21886427700519562, -1.505145788192749, -2.368178129196167, 0.6750055551528931, -0.08876167982816696, -0.31615352630615234, 0.6401785612106323, 1.48623788356781, 0.974536120891571, -0.22602617740631104, 0.18631786108016968, -2.1355907917022705, -0.009869401343166828, 3.1724627017974854, 1.6390376091003418, -2.4896535873413086, -0.008050450123846531, 1.595649003982544, 0.10646692663431168, -0.23722407221794128, -1.5502382516860962, -0.7878327965736389, 1.154198169708252, 0.2452869564294815, -1.212295413017273, 0.8457715511322021, -1.711809754371643, -1.7348428964614868, 1.257930040359497, -0.2515696585178375, -0.4821399748325348, -0.8300464749336243, -2.6244871616363525], [0.9495531320571899, 1.3691647052764893, 1.3327677249908447, 0.4461807608604431, 1.7014294862747192, -0.0628456324338913, -0.9192131161689758, 1.2958757877349854, 0.3075587749481201, 0.24356257915496826, -0.9288844466209412, -0.6802555918693542, 2.1090872287750244, -0.20406755805015564, 1.5091999769210815, -0.7030417919158936, -0.9501240849494934, -0.8932791352272034, 1.4302964210510254, -0.9672317504882812, 2.0821125507354736, 0.20417700707912445, -0.3623010218143463, -1.8338490724563599, 0.690024733543396, -0.21774981915950775, 0.4139285087585449, -0.3145674765110016, -0.3922072649002075, 1.1726162433624268, 0.7118504047393799, -0.44553956389427185, 1.0817553997039795, -0.6789945960044861, 1.4163655042648315, 0.5067110061645508, -2.119119644165039, 0.17929264903068542, -0.014302127994596958, -0.9152926802635193, -0.31961193680763245, 1.1266748905181885, -0.7275444865226746, -1.5103323459625244, -0.5838260054588318, -0.17270196974277496, 0.8853003978729248, 0.20768079161643982, 0.18605473637580872, 0.9948927760124207, 0.05434775352478027, -0.1427846997976303, 1.032927393913269, -0.5009063482284546, 0.7160068154335022, -0.8769205808639526, 1.4474605321884155, 1.4578319787979126, 0.6081525683403015, 1.6338149309158325, 1.5654348134994507, -0.6436653733253479, 1.6495743989944458, -0.026735054329037666, -0.4736034870147705, 0.9666875600814819, -0.6879278421401978, -0.34887439012527466, -0.6953351497650146, -0.23353461921215057, 0.913829505443573, -1.0193573236465454, -1.622448444366455, 0.15661759674549103, 1.0676137208938599, -1.1270171403884888, -0.1624022275209427, 0.3491376042366028, -1.872999668121338, 0.1913483887910843, -1.151841402053833, 0.00693527702242136, -0.337911456823349, 1.0839842557907104, 0.29346713423728943, -1.466598391532898, 0.9565587043762207, 1.2808729410171509, -0.5095275044441223, -0.052242714911699295, -0.15327239036560059, 0.47860097885131836, 1.5785466432571411, -0.030062586069107056, -0.16546925902366638, -0.3037468194961548, 0.8664003014564514, 0.936529815196991, 1.2582569122314453, 1.0270395278930664, -0.009777755476534367, 0.2160218209028244, 0.7157925963401794, 0.4177480638027191, -2.1995112895965576, -0.4193161725997925, 0.8474350571632385, 1.7223058938980103, 2.0151689052581787, -0.27245497703552246, 0.6649518013000488, 0.5050366520881653, -0.7890710830688477, -1.7421915531158447, 1.2439014911651611, 0.4457254707813263, -1.1294232606887817, -0.31692183017730713, -0.35939598083496094, -0.262626588344574, 0.8547948002815247, -0.22884730994701385, -0.4576524794101715, -0.47427302598953247, 1.7238996028900146, 2.1804137229919434, -2.3392815589904785, -1.042074203491211], [0.23960675299167633, -0.6152375340461731, -0.9664722084999084, -1.6654274463653564, -0.445173054933548, -0.5424310564994812, 0.34659695625305176, 0.4873717725276947, 2.6248674392700195, 2.158332109451294, -1.5015279054641724, -0.4308764934539795, -0.18961788713932037, -0.39295506477355957, 0.8017473816871643, 0.21562747657299042, 0.17920060455799103, -1.033545732498169, 1.9675424098968506, 0.6530972123146057, -0.23074205219745636, -0.9113009572029114, -1.6163334846496582, 1.4177857637405396, 0.29681578278541565, 0.5683087706565857, -0.7766745686531067, -1.6814483404159546, -0.9605131149291992, 0.9326037764549255, -1.1230504512786865, 0.17294563353061676, -1.088677167892456, 1.450402855873108, -0.1733570396900177, -1.1915826797485352, -1.1067683696746826, 0.10057278722524643, 0.16335327923297882, 0.09772155433893204, -1.5721652507781982, -1.8347702026367188, -0.04925217479467392, -0.36305689811706543, 1.8719062805175781, 0.22387075424194336, 0.6578496098518372, -1.769918441772461, 2.3109655380249023, 0.7516200542449951, -2.4807379245758057, -0.2972213923931122, -3.0475986003875732, 2.3431406021118164, -0.08922535926103592, -0.10203792154788971, -0.2574179768562317, 0.6231669783592224, 0.42940381169319153, -0.40458789467811584, 0.613111138343811, -1.0489206314086914, -0.6361834406852722, -0.10255461186170578, 2.112416982650757, 1.2511454820632935, -0.3656976521015167, 0.2826787233352661, -1.0376427173614502, -0.01484991516917944, 0.8450416922569275, 1.8320727348327637, 0.11643943935632706, 0.9612916707992554, -0.44257405400276184, 0.2147628664970398, 1.4893290996551514, -0.7745238542556763, 1.3600794076919556, 0.1531280130147934, 0.11863067746162415, -0.060866642743349075, 1.6252241134643555, 0.8790664672851562, 0.14699746668338776, 2.1101596355438232, -0.7334734797477722, -0.45692503452301025, -0.018842890858650208, 0.3663935661315918, 0.3031974136829376, 0.38493672013282776, -0.6802968978881836, 0.3404446840286255, -0.7750133275985718, 1.6208093166351318, 0.13605985045433044, 1.4064881801605225, 2.11360239982605, -0.17686399817466736, -0.43124186992645264, 0.5971010327339172, -1.007601261138916, 0.7041069269180298, -1.1406642198562622, 0.5973278880119324, 0.11482415348291397, 0.263935923576355, 1.5213440656661987, -1.9695407152175903, -1.621958613395691, 0.8867729902267456, 0.0035071736201643944, -0.593214750289917, 0.5965838432312012, 0.7282432317733765, -1.4243751764297485, -1.1124308109283447, -2.1553635597229004, -0.9880403280258179, 1.8883720636367798, 0.5148816108703613, -0.8105604648590088, -0.4111218750476837, -3.047274351119995, 0.8228339552879333, -0.7010340690612793, 2.573596954345703], [-0.44162020087242126, 0.952438473701477, 1.2222639322280884, -0.007629367522895336, 0.08407045900821686, 0.8380585312843323, 0.6524472236633301, 1.477612018585205, -0.08048328012228012, -0.16016481816768646, -0.022177569568157196, 1.740366816520691, 0.614599883556366, 1.9921320676803589, -0.7400598526000977, -0.24654649198055267, 1.2773743867874146, -1.9955464601516724, -0.49276021122932434, 0.7402598857879639, -0.6055814623832703, 0.33142682909965515, 0.7124477028846741, 0.413863867521286, -0.6174044609069824, -0.9340152740478516, 0.9303035736083984, 1.179067850112915, -1.207964539527893, 0.4261133074760437, -0.277815043926239, -1.0260672569274902, -0.282108336687088, -1.6619081497192383, -0.23519361019134521, 0.7157169580459595, -0.11968476325273514, -0.9778620004653931, -0.6788640022277832, 0.6817455887794495, -1.429071307182312, -0.8167498707771301, -2.3248960971832275, 0.9709397554397583, 0.26986265182495117, 0.10961484163999557, 0.7727377414703369, -0.33129093050956726, 1.851515531539917, 1.9440597295761108, 0.2726268172264099, -0.6387683153152466, 0.8868075609207153, -0.5010702610015869, -0.41086122393608093, -0.8423647284507751, -0.05351792648434639, 1.1709141731262207, -0.44659316539764404, -1.1272857189178467, -2.288489580154419, 0.3852580785751343, 1.1783498525619507, -0.17867626249790192, 0.5367128252983093, -0.20524467527866364, 0.37550824880599976, 0.5844918489456177, -0.9795295596122742, 1.232434630393982, 1.1606992483139038, -1.4122623205184937, 0.9419257640838623, 1.6440201997756958, -1.850132703781128, -0.7002547383308411, 0.058661751449108124, -2.9766054153442383, 1.257513165473938, -0.9734837412834167, -0.17259758710861206, 0.16242007911205292, 0.7771929502487183, 1.3373697996139526, 1.035765290260315, -1.3022842407226562, -0.212471604347229, 0.9056376218795776, -1.1412200927734375, 0.03526308014988899, 1.825103998184204, 0.6447955369949341, 0.7350517511367798, 0.17599502205848694, -0.5846333503723145, 0.5146940350532532, -0.9596915245056152, 1.0696436166763306, 1.085042119026184, 1.0063755512237549, -0.6418564319610596, 0.9119061231613159, -0.48101094365119934, 0.12531805038452148, -1.1103777885437012, -0.7421097159385681, -0.1545930653810501, -0.15296481549739838, -1.0733473300933838, 2.1558029651641846, -0.4841301143169403, -0.5450036525726318, 1.7312716245651245, -0.42816153168678284, 1.2538824081420898, -0.11693640798330307, 1.5005234479904175, -0.32424792647361755, 0.17991819977760315, -0.5709139108657837, 0.006142234895378351, 0.3304730951786041, -1.0129863023757935, 1.5401840209960938, 0.594457745552063, -0.0029624979943037033, 0.5405033826828003, -0.359497994184494], [-0.9250475764274597, 1.5803159475326538, 0.11538193374872208, -0.9227914810180664, -0.9523919820785522, 0.880821704864502, -0.10220129042863846, -0.6236928105354309, 1.3115086555480957, 0.03219499811530113, 0.6351562738418579, 0.4411297142505646, 1.3544466495513916, 0.986649215221405, 0.07899434119462967, -1.183984398841858, -0.7990521192550659, -1.3738183975219727, -0.7298614978790283, 0.3168398439884186, 0.8314837217330933, -0.6248770952224731, -2.1506905555725098, -0.3938571512699127, -0.47350001335144043, -1.0601856708526611, -0.36732515692710876, -1.0074677467346191, 0.20375420153141022, 0.03282282128930092, -0.5289581418037415, -2.0009264945983887, -0.8150928616523743, 1.0634022951126099, 0.006949800066649914, -0.04587843269109726, 1.4715955257415771, -0.4162580370903015, 0.29940065741539, 0.39574047923088074, -1.2600966691970825, -0.17852455377578735, -0.06143319979310036, -0.2834300696849823, 0.058976370841264725, -1.3878264427185059, -0.05506980046629906, -0.11022411286830902, 1.4450525045394897, 0.8959226608276367, 0.5184774994850159, -1.037548303604126, -2.4052975177764893, 0.9799063801765442, -0.0967104434967041, 0.5017842054367065, 0.15062089264392853, -1.1540677547454834, -0.36395159363746643, 0.15721337497234344, -0.06050986424088478, 1.5543104410171509, 0.3299991488456726, -1.9274100065231323, -0.547058641910553, -1.3835244178771973, 1.1503225564956665, 0.329965740442276, 0.765605092048645, -1.870292067527771, 0.21276451647281647, 0.9294165968894958, 0.598129153251648, 0.442905068397522, -1.1176894903182983, -0.705522358417511, 1.4442259073257446, 0.1772954761981964, -1.4833523035049438, -0.011262478306889534, -0.2560088634490967, 0.08770647644996643, -0.4407743215560913, 1.3100061416625977, 0.37039729952812195, 1.2271634340286255, -0.9470820426940918, 1.1172443628311157, -0.30599671602249146, -1.5484168529510498, 0.731652557849884, -1.023561716079712, 1.8593238592147827, 0.8678382039070129, 0.9554557204246521, -0.25681358575820923, 2.1129696369171143, -2.030477523803711, 0.2079962193965912, -0.7586932182312012, -0.14664076268672943, 1.1531250476837158, 0.06099426746368408, 0.05445941537618637, 0.717889666557312, 0.9559429883956909, -0.02044537477195263, 0.8077981472015381, -2.053905487060547, 1.1667569875717163, -0.001993929734453559, 1.58942449092865, 1.2294068336486816, 1.6623475551605225, 2.0859365463256836, 0.46028298139572144, 1.690248966217041, -0.31809377670288086, 0.6952238082885742, 1.1280591487884521, 1.2583779096603394, -0.01918487623333931, -0.4505355954170227, 1.0246491432189941, -0.4277462065219879, -0.1488601416349411, -0.22444775700569153, 1.7957566976547241], [-0.6158627271652222, 0.08213627338409424, -0.5248544216156006, 1.5445632934570312, -0.5292037129402161, -0.5358462333679199, 0.8313878774642944, -1.5085991621017456, 1.085752010345459, 0.7902988791465759, -0.33906134963035583, -1.9123982191085815, -1.2908353805541992, -0.15344232320785522, -0.5326337814331055, -1.102625846862793, 0.019773265346884727, 1.920215368270874, -0.8208668231964111, -0.8372436165809631, -0.5536622405052185, 0.20831996202468872, 1.241049885749817, -0.2479529231786728, 1.8141759634017944, 0.3603731393814087, -1.509505271911621, 0.2768888473510742, -0.6667258739471436, 1.8313673734664917, 0.5000348091125488, -0.018428664654493332, -0.5900256633758545, 0.16271741688251495, 1.3837788105010986, 1.4716839790344238, -0.9208126664161682, 0.04381995648145676, -1.7428393363952637, -0.38844552636146545, 0.8208174109458923, -0.6537169218063354, 1.124982476234436, 0.1401367038488388, -0.45780444145202637, -1.5472831726074219, -0.5854416489601135, -0.953715980052948, -0.7371312379837036, -0.6134194135665894, -0.4764387905597687, -2.1487371921539307, 0.10300252586603165, 0.09405779838562012, 0.3849848210811615, 1.163347601890564, 1.553330421447754, -0.7061554789543152, 0.4719133973121643, -1.9580093622207642, 0.31315872073173523, -0.6267673373222351, 1.2088806629180908, 1.4552274942398071, 0.5603217482566833, -0.11009531468153, 1.1992733478546143, -0.3187260329723358, -0.37145882844924927, 0.48701420426368713, 0.9652803540229797, -0.23950663208961487, 1.3168764114379883, -1.2575597763061523, 1.5994399785995483, 1.2669286727905273, 1.5228761434555054, -0.682232677936554, -0.5268694758415222, 0.9791333079338074, -0.4476451575756073, 0.8843319416046143, -0.8190527558326721, -0.558176577091217, -0.3342307507991791, -1.1464695930480957, 0.5249307751655579, 1.350592017173767, -0.2683373689651489, -1.1430294513702393, -0.48271241784095764, 0.22190816700458527, -0.4040111303329468, 1.1136711835861206, -1.0230532884597778, -0.43197521567344666, 0.5090075135231018, -0.006321679335087538, -0.9123170971870422, 2.2447569370269775, -0.13722611963748932, -0.7304970622062683, -0.07567824423313141, 0.3748568594455719, 1.4966996908187866, 0.5567682981491089, 0.14599673449993134, -0.034658655524253845, 0.032393161207437515, -0.8413712978363037, -0.19909116625785828, 0.27692148089408875, -0.7072840929031372, 0.5098819732666016, 0.36434122920036316, -0.8389787673950195, 0.9336262345314026, 1.4750877618789673, 1.9846745729446411, 1.5607373714447021, -0.5363287925720215, 0.44954803586006165, -1.4048079252243042, 0.46406805515289307, -0.8071666359901428, 1.4540479183197021, 0.004474736750125885, 0.01704087108373642], [-1.9154280424118042, -1.4867900609970093, 2.5604937076568604, -0.4764433205127716, -0.34543749690055847, -1.6736098527908325, 0.3846500515937805, 1.205112099647522, 0.94817715883255, 1.4255303144454956, 0.2681502401828766, -1.361242413520813, 0.06453206390142441, -0.5773674249649048, -0.014183645136654377, -0.09629493206739426, 2.6946752071380615, -0.43546295166015625, 0.2180798053741455, -1.4241571426391602, 0.5760035514831543, 1.2106070518493652, -0.46306437253952026, -2.023083209991455, -1.572216510772705, -1.6211028099060059, 0.6223114132881165, -0.0174131877720356, -1.9419958591461182, -0.4746829569339752, 1.373108148574829, 1.6823346614837646, 1.4212251901626587, 0.8912345767021179, -1.4671229124069214, 1.6617929935455322, -0.988439679145813, 0.011661427095532417, 0.5496955513954163, 0.9269899129867554, 2.57002592086792, -0.48461437225341797, -0.7036548852920532, -0.00333771831355989, 1.006098985671997, -0.6341367959976196, -2.056638479232788, -0.4441869556903839, -1.176256537437439, 0.33797696232795715, -1.0756251811981201, 0.5733353495597839, 0.25649601221084595, -0.0971844345331192, 0.22733981907367706, -0.1562730222940445, 0.2573085427284241, -0.3400571346282959, -0.8090924620628357, -1.281052589416504, -0.7363096475601196, 0.16771842539310455, 0.014554009772837162, 0.4115980863571167, 0.9561557173728943, 0.8342071175575256, 1.3493635654449463, 2.762047052383423, -1.8205119371414185, -0.6499192714691162, 0.05259086191654205, -1.2310051918029785, -0.4527374505996704, -0.5159175395965576, 2.6389284133911133, 0.8068347573280334, 0.26981931924819946, -0.6833754777908325, -0.03829265758395195, 0.32355132699012756, -1.6814316511154175, -0.9145324230194092, 1.2563471794128418, -1.030724287033081, -0.45916545391082764, 0.015552897937595844, 0.7294397354125977, -1.8007144927978516, 1.2397617101669312, -0.5250563025474548, -0.6371228098869324, 0.6372992992401123, 0.15242967009544373, -1.1605812311172485, 0.7620553970336914, 0.14939478039741516, -0.4459996521472931, -1.0344411134719849, -1.4472014904022217, 1.619306206703186, -0.9258502125740051, 0.7272728085517883, -0.01985090784728527, 0.7250662446022034, 1.3549222946166992, 1.374091386795044, 0.7247408032417297, -0.8291018605232239, 0.07413842529058456, 0.40205028653144836, 0.2887018918991089, -3.071220636367798, 0.49244287610054016, -1.7724145650863647, -1.781935453414917, -0.22697876393795013, -0.9087541699409485, -0.3783750534057617, 0.3027366101741791, -0.8793888688087463, 1.0694117546081543, -1.7407245635986328, 0.34074440598487854, -0.5494018793106079, 0.6156412363052368, -1.0926202535629272, 1.0699833631515503, 0.6498395204544067], [0.5261813998222351, 0.034540556371212006, 0.037567831575870514, 1.485257863998413, 0.3562881052494049, -0.1311141699552536, -0.873170018196106, -1.0625661611557007, 1.0895605087280273, 1.203061580657959, 0.2835710942745209, -1.3274692296981812, 0.2482917308807373, 1.2979899644851685, 0.21648110449314117, -0.9219633340835571, 0.35606783628463745, -1.4418532848358154, 0.7990110516548157, 0.15369105339050293, 0.05526120215654373, 1.404484748840332, 0.7458359599113464, -0.2544234097003937, 0.5720862746238708, 0.45055216550827026, 0.9786557555198669, -2.2869176864624023, -0.09464766085147858, 0.8817567229270935, 0.4542013108730316, -1.6453545093536377, 1.0883617401123047, -0.027919145300984383, -0.4874114692211151, 1.2470307350158691, 0.6723790764808655, 1.2418550252914429, 0.43044304847717285, 0.6009923815727234, 1.1497458219528198, -0.12591741979122162, -1.1790950298309326, 2.1409926414489746, 0.012126343324780464, -0.49056339263916016, -0.007167846895754337, 1.765813946723938, -2.0561418533325195, -0.2378867268562317, -0.9132682085037231, 0.8481668829917908, 2.1884255409240723, 0.4568788707256317, -1.5560282468795776, -1.4660594463348389, -1.8342278003692627, 0.7147964835166931, -1.0747605562210083, 0.20980960130691528, 0.4578796923160553, 0.6977953314781189, 0.9982296228408813, -1.1162776947021484, -1.60544753074646, 0.5941896438598633, 2.1166110038757324, 1.3174376487731934, -1.1437122821807861, -1.237746238708496, -0.5539114475250244, 0.28326845169067383, -1.1083965301513672, 0.7902759313583374, -1.1331008672714233, -0.616969883441925, -1.1606277227401733, -0.6286569833755493, 0.170012429356575, 0.2558774948120117, 0.9920676350593567, -1.6073580980300903, 1.0460312366485596, 0.3347693681716919, -0.17954318225383759, 0.09439247846603394, -0.659669816493988, 0.7190518379211426, -0.8705541491508484, 1.2833023071289062, -0.7097665071487427, -1.645776629447937, -0.37236639857292175, -0.20069104433059692, 0.09503058344125748, 1.8404529094696045, 0.07410148531198502, 0.9793491363525391, -0.46739065647125244, 0.43456757068634033, 2.6837425231933594, -0.2362535148859024, -1.0462284088134766, 0.8046846389770508, 1.9124826192855835, -1.648646593093872, -0.25135377049446106, -1.2625099420547485, 0.5094383358955383, -0.708020806312561, 0.6193879842758179, 0.026201436296105385, 0.7715908288955688, -0.10857532173395157, -1.6365938186645508, 0.48003703355789185, 0.4515257179737091, 1.7426255941390991, -1.7691709995269775, -1.7318332195281982, -0.8922529816627502, -0.17222340404987335, 0.09498261660337448, -0.889493465423584, -0.39613744616508484, 0.5394115447998047, -1.0561007261276245, 0.10896492004394531], [0.08848920464515686, 0.37930068373680115, -0.303292453289032, -0.0609104298055172, 0.6882441639900208, 0.4733735918998718, 0.7450793385505676, 1.5536372661590576, -1.1527371406555176, 2.17441987991333, 0.535245418548584, -0.9302583932876587, 0.6373268365859985, 0.5102418661117554, -0.3946390151977539, 1.1470626592636108, 2.1483957767486572, 1.699509859085083, -1.3705333471298218, -0.6569100022315979, -1.0293614864349365, 0.5164161324501038, 0.25144749879837036, 0.5879125595092773, 0.7263184785842896, -0.4802209734916687, 0.3525764048099518, 0.054261963814496994, -2.106501817703247, -1.5743964910507202, 1.274857521057129, -0.9839341640472412, -0.03542608395218849, -0.9391441345214844, -1.6162234544754028, -0.30009937286376953, 1.4437508583068848, 1.1120816469192505, 2.0616002082824707, -0.8894633054733276, -0.13254320621490479, -0.49898216128349304, 0.7240588068962097, 0.7122242450714111, -0.08166521042585373, 0.3570009768009186, -0.5419843792915344, -1.225123405456543, 0.09548777341842651, -0.4383811950683594, -0.9086015224456787, 0.8840702176094055, 1.757115364074707, -1.8452824354171753, 0.6374784708023071, 2.2538764476776123, 0.9176099896430969, 2.0154595375061035, -0.1286558359861374, 1.4912362098693848, 1.3515982627868652, -0.6927669644355774, -0.3343104124069214, -0.705323338508606, 2.367678642272949, 1.549201488494873, -0.5160741806030273, 0.4723496437072754, -1.2994338274002075, -0.11279390007257462, -1.7907447814941406, 0.5876505970954895, -0.29561248421669006, -0.17489056289196014, -0.059161849319934845, -0.6347610354423523, 0.7419407963752747, -1.6353189945220947, 0.06840430200099945, -1.3892408609390259, -0.22136737406253815, -0.6399743556976318, 0.8865654468536377, 0.4895544648170471, -0.9915675520896912, 0.8798362612724304, -1.351435661315918, 0.7539495825767517, 1.168888807296753, 0.01218224037438631, -0.5391069650650024, -0.033084988594055176, 0.31325867772102356, -0.06494281440973282, 1.1132713556289673, -0.19508133828639984, -0.5022904872894287, 1.4079211950302124, 0.06786801666021347, -1.3912802934646606, -0.017179543152451515, -0.6845048069953918, 0.45781606435775757, -1.0264980792999268, 0.9420714974403381, -0.0037377835251390934, 1.4778521060943604, 0.27323436737060547, 0.675719141960144, -0.10409756004810333, -0.5252934694290161, 1.4576950073242188, -0.9015889167785645, -0.29392489790916443, 1.0449217557907104, 1.2800509929656982, 0.38817888498306274, 2.1230223178863525, 1.9321907758712769, -1.0320723056793213, -0.8696765899658203, -2.3913943767547607, -0.03773350268602371, 0.26285719871520996, -1.4012866020202637, 0.004064739681780338, 2.157578468322754, 0.09795279055833817], [-1.7202470302581787, 0.6565225720405579, -0.4923691749572754, 0.020061494782567024, -1.223546028137207, -0.003689279081299901, -0.23370607197284698, 0.5521968603134155, -0.26731210947036743, 0.3676080107688904, -1.4129377603530884, -1.507748007774353, 0.9805610775947571, -0.9099335670471191, -0.10934196412563324, 0.6599475741386414, -0.9203002452850342, -0.008328935131430626, -0.5541449189186096, -0.0232948437333107, 0.5345120429992676, -1.110609531402588, -0.8144350051879883, 0.8004288673400879, -1.3196028470993042, 0.4134089946746826, -1.6783291101455688, 0.05082755908370018, 0.6265084147453308, -1.0821266174316406, 0.6853083372116089, -1.0050551891326904, 0.8160513639450073, 0.28124597668647766, -1.2335491180419922, 1.8841928243637085, 0.37914782762527466, 4.164571762084961, -0.16821759939193726, 0.9804775714874268, 0.8572930097579956, -0.6350287199020386, -0.21887923777103424, 0.33494213223457336, -0.6302425861358643, 1.0013786554336548, 1.1528205871582031, 1.057813286781311, 0.3437533378601074, -0.2581164240837097, 0.8167206048965454, -0.24562132358551025, -1.0904390811920166, 0.43057355284690857, 0.23237109184265137, -0.667636513710022, 0.9768046140670776, -0.6932281255722046, -1.4081610441207886, -1.7176164388656616, 0.3739975094795227, -0.004981375765055418, -0.3554430902004242, 0.9513135552406311, -1.133936882019043, 0.8117672801017761, 0.32935425639152527, 1.4878003597259521, -1.0534437894821167, -0.4491927921772003, 0.011191566474735737, 1.4776208400726318, -0.46827948093414307, -1.99833083152771, -0.5822295546531677, -0.4100905656814575, -0.19070342183113098, -0.9392359852790833, 1.2642626762390137, 1.1246494054794312, 0.49196097254753113, -0.8056202530860901, -0.5019873380661011, 0.35161247849464417, 0.32403674721717834, 0.059678953140974045, -0.8955587148666382, -0.5742112994194031, -1.2289867401123047, -0.08743126690387726, 0.18873324990272522, -0.5479856729507446, 0.5186260938644409, -0.3987632691860199, 0.03674333915114403, 0.5345502495765686, -0.033188894391059875, 0.07001598924398422, -0.5940873026847839, 0.3799467086791992, 0.03813447058200836, -0.10499396920204163, -1.0680662393569946, -0.2791730761528015, 0.2978590726852417, 0.2180662453174591, -0.08487989753484726, 0.48896241188049316, 0.7894688248634338, -2.50215482711792, -1.0466387271881104, 0.8505341410636902, -1.110116958618164, -2.041841983795166, 0.5869250893592834, -1.246596097946167, -0.3182874321937561, 0.3557535707950592, 0.11388064920902252, 0.5854392647743225, -1.059535026550293, 1.7817953824996948, 1.1822603940963745, 1.78901207447052, -0.7178133726119995, 1.312781810760498, -1.91548752784729, -0.8705666065216064], [0.11068614572286606, 2.301638603210449, 0.8000109195709229, -0.44352442026138306, 0.5858115553855896, 1.4885684251785278, -0.3382718861103058, 0.7231788039207458, 0.41995084285736084, 0.17804457247257233, -0.03076324239373207, 0.6897040605545044, 0.34061357378959656, -0.7490102648735046, -0.38328200578689575, -0.3680680990219116, 0.8537282347679138, -2.5946834087371826, -0.07224888354539871, -1.0174856185913086, 0.9185577034950256, -0.2063307911157608, 2.2879130840301514, 0.9261802434921265, -0.3025818169116974, 1.1934525966644287, -0.2591366767883301, 0.2063944935798645, -0.8408004641532898, -0.35725387930870056, -1.5439081192016602, -0.9252646565437317, -0.6629365682601929, 1.01939857006073, -0.7684518098831177, -1.7083948850631714, 1.5569641590118408, -1.0795375108718872, -1.1038285493850708, 2.6261327266693115, 0.9146869778633118, -0.29389938712120056, -0.15828083455562592, 0.2894916534423828, 0.39372196793556213, -0.6741621494293213, -0.10345837473869324, 0.307563841342926, 1.0378451347351074, 1.1754902601242065, -0.5729215145111084, 0.9107866883277893, 0.7767149806022644, -0.3171465992927551, 2.0749309062957764, 1.6954163312911987, 1.0800354480743408, -0.4182994067668915, -0.00600137235596776, -0.6031744480133057, -0.4153141379356384, 1.2160207033157349, -0.5209140181541443, -0.06425425410270691, 0.2445671707391739, -1.1659715175628662, 2.3931186199188232, -0.1806894838809967, 0.3160446286201477, 0.7020689845085144, 0.6630762219429016, 2.330122232437134, -0.3998297452926636, -0.10364892333745956, 0.01722296141088009, -1.490812063217163, 1.9068185091018677, 0.6076216101646423, -0.28187575936317444, -0.6143874526023865, 1.642767071723938, -1.1113067865371704, 0.09652268886566162, 0.8523145914077759, 0.5495249032974243, 0.10916599631309509, 1.8534176349639893, 0.36664456129074097, -1.5322779417037964, 1.612144112586975, 0.04232889786362648, 0.1731029897928238, 0.1711825579404831, 1.925477147102356, 1.4735431671142578, -0.27755045890808105, -1.2594908475875854, -1.8022817373275757, -0.14053010940551758, -0.5974140167236328, 1.1136064529418945, 1.3392994403839111, 0.5052404403686523, 1.2037378549575806, -1.8395962715148926, -0.45087990164756775, 0.4111126661300659, 0.4958760738372803, 1.1491899490356445, 0.4526143968105316, -1.203431248664856, -0.021506819874048233, 1.1507972478866577, 0.02870236337184906, 0.6459420919418335, -1.2463473081588745, -1.1693717241287231, -0.43919816613197327, -0.7455887198448181, -0.7713250517845154, 1.48738694190979, -0.14021660387516022, 0.46745336055755615, 0.04145461320877075, -1.1160013675689697, 0.9290597438812256, 0.4415043592453003, 1.3418036699295044], [-1.0632296800613403, -0.9518004655838013, 0.2508580684661865, -0.8154480457305908, 0.2674594223499298, 0.2355409860610962, -0.24510037899017334, 0.03295554593205452, 0.4733724296092987, -2.2348082065582275, 1.421452283859253, -0.4939000904560089, 0.014676036313176155, -0.1392541527748108, 0.6385570764541626, 0.7139182686805725, -0.9612033367156982, 0.054017841815948486, 1.9601635932922363, -1.541282057762146, 0.49493908882141113, 1.2869858741760254, 0.5204080939292908, -0.817321240901947, 0.9976617097854614, 0.21552139520645142, 0.49238336086273193, 0.3745162785053253, -0.875474214553833, -1.0129886865615845, 0.9947607517242432, -0.07884581387042999, 1.197432518005371, 1.1746584177017212, -0.5309704542160034, 0.9897462725639343, -0.7655070424079895, 0.5649265050888062, -0.2351846992969513, 0.10105999559164047, -1.8296611309051514, -0.340378999710083, -1.3517218828201294, 0.7603275179862976, -0.24875086545944214, -0.45682579278945923, 0.6976255774497986, -1.3163856267929077, -0.1628498136997223, 1.388291597366333, 1.4289708137512207, 0.05894593894481659, -0.05898712947964668, 0.20934031903743744, -0.5783843994140625, 1.7709599733352661, 0.36272066831588745, -0.21565453708171844, -0.7342293858528137, 1.7858439683914185, -0.7813736796379089, 0.04341680556535721, 0.9822879433631897, -0.08571161329746246, -0.5399422645568848, -0.5093246698379517, -0.16166472434997559, -2.0562243461608887, -0.5307754278182983, -1.4138864278793335, 1.7402114868164062, 0.947860598564148, -0.2416798323392868, 0.46792447566986084, -1.5202569961547852, 0.7122060060501099, 1.0278761386871338, 0.11929920315742493, 0.9244219064712524, 1.7503753900527954, -1.1731226444244385, -0.614656388759613, -0.682140588760376, 0.07738985121250153, 0.5638154745101929, -0.6608991622924805, 1.0934069156646729, -0.3881983458995819, -0.17703713476657867, 0.45743700861930847, -0.5451791286468506, -0.3465569019317627, -0.29507288336753845, -0.8436191082000732, -0.23931194841861725, -0.09385974705219269, -0.4532855451107025, -1.1963976621627808, 1.365646481513977, -0.012005474418401718, 1.4182546138763428, 0.62876957654953, -1.106654405593872, 0.747087299823761, 1.5618526935577393, -0.9154189825057983, 2.271074056625366, 0.2056931108236313, 0.3166356682777405, 0.48534268140792847, -0.5491438508033752, 0.23860739171504974, 1.1037739515304565, -0.5214847326278687, -0.2230728417634964, 0.914897084236145, -1.6179934740066528, -0.0030691404826939106, -1.867443561553955, 0.0429094024002552, 0.2741139531135559, 1.2448949813842773, 0.5420441627502441, -0.5212832093238831, 0.5732113718986511, 1.5289289951324463, 1.221048355102539, -1.118248701095581], [1.3513946533203125, -0.4010506272315979, -1.8127033710479736, -1.063232183456421, 1.115753173828125, 0.40831300616264343, -0.057170696556568146, 0.095310278236866, -0.06694603711366653, 0.9054879546165466, -0.14423160254955292, -0.6872647404670715, -0.5238622426986694, -0.04004353657364845, 0.010555674321949482, -0.12783443927764893, -2.1847424507141113, 0.4889805316925049, -0.9924360513687134, -1.2379121780395508, 0.7746155261993408, 1.1187777519226074, -1.1034811735153198, 0.2767578065395355, -0.7990652918815613, -1.1446893215179443, -0.21905963122844696, -1.338651180267334, 2.287782669067383, -0.7271995544433594, -0.4459283649921417, 1.649180293083191, -0.8157678246498108, 1.4696626663208008, -0.27349767088890076, 0.7877621650695801, -0.20269054174423218, 1.1736063957214355, -2.4526658058166504, -1.8983174562454224, 0.5754233598709106, 0.7423513531684875, 0.2702120542526245, -1.4272947311401367, -2.1549956798553467, 0.7464961409568787, 0.5870116353034973, -1.5193246603012085, -0.35502204298973083, -0.6281874775886536, 0.1134805977344513, -0.26656022667884827, 0.3589807152748108, 1.0840774774551392, 0.6337548494338989, 0.6976792216300964, 0.3892231285572052, -0.550303041934967, -1.1857383251190186, 0.8221402764320374, 0.13727974891662598, 1.2328242063522339, 0.07962944358587265, -0.2872527241706848, -0.2668415307998657, 0.20925311744213104, 0.7315426468849182, 0.22033244371414185, -1.2440282106399536, 0.7541980743408203, 1.4383392333984375, -0.9076144099235535, 0.40599432587623596, 0.1924566775560379, 1.3955212831497192, -1.0575213432312012, -1.1908295154571533, -1.1799968481063843, 0.39493533968925476, -0.7736594080924988, 0.3911643624305725, -1.3991445302963257, 0.013135538436472416, 0.5068346858024597, 1.2820441722869873, 1.3884936571121216, -1.2010570764541626, -1.8108584880828857, 0.5727859139442444, 0.1603269726037979, -0.7794339060783386, -1.105899691581726, -0.37093549966812134, -0.02665577083826065, -0.23241852223873138, -0.7252698540687561, -0.7228087186813354, -0.7156789302825928, 0.09234879910945892, -0.28371384739875793, -0.3635920584201813, 0.6891435384750366, -0.6097677946090698, -1.1786011457443237, -0.15384148061275482, 1.4065203666687012, -0.2975209951400757, 0.8097922801971436, -0.8158605098724365, 0.32206296920776367, 0.8072841167449951, 0.42542171478271484, 0.7348028421401978, 0.3776165246963501, 0.7107653021812439, 2.3883702754974365, 0.6690073609352112, -1.7175418138504028, -0.33982646465301514, -0.8754248023033142, 0.5957417488098145, -1.166879653930664, 0.09458966553211212, -0.5504216551780701, -0.6106564998626709, -0.9602755308151245, -0.026817727833986282, 0.3002031445503235], [0.08113507926464081, -1.2665088176727295, -0.38007351756095886, 1.403639793395996, -0.23528271913528442, 0.43970149755477905, 0.6474074721336365, -2.0817856788635254, -2.6291263103485107, 1.1084092855453491, 0.6008807420730591, -0.016829093918204308, 0.12841898202896118, 1.7154730558395386, 0.2776632308959961, -0.7238329648971558, -0.01838293857872486, 1.7114713191986084, 2.702859878540039, 0.21830131113529205, -0.5168461799621582, 0.02158472314476967, 1.1192331314086914, 0.5496529340744019, -0.9820302724838257, 2.254892349243164, -1.1668791770935059, 1.5522258281707764, -0.9034914374351501, -0.09226008504629135, 0.1717681735754013, 1.5842268466949463, -0.24503503739833832, 0.7009538412094116, 0.543490469455719, 0.704024076461792, 0.41080206632614136, -1.0086885690689087, -1.4819650650024414, -0.13019034266471863, 0.05035964027047157, -0.3511718809604645, 0.17649920284748077, 0.6516426205635071, 0.432807594537735, 0.20519116520881653, -0.046390749514102936, 0.2970466911792755, -1.2385715246200562, 0.03330395370721817, -0.717847466468811, -0.18176861107349396, -1.3153234720230103, 1.5572130680084229, -0.1277778148651123, 0.7996224761009216, -0.6181573271751404, -1.4143304824829102, 0.0772302895784378, 0.7759631872177124, -0.5109527111053467, -1.788025140762329, 0.6218525171279907, 0.36005279421806335, -0.43360090255737305, 0.48048990964889526, 0.10275045782327652, -2.0225446224212646, -0.010765613988041878, 0.4181451201438904, 0.5163038372993469, 0.06237312778830528, 1.0996787548065186, -1.9819306135177612, 0.8338620066642761, 0.2785916328430176, 1.2374969720840454, 1.0877792835235596, -1.0123724937438965, -0.21266469359397888, -1.3038456439971924, -0.13673901557922363, 2.2593281269073486, 1.5742225646972656, -0.5121874809265137, -0.3265278935432434, 1.5285305976867676, -1.6697148084640503, -0.07831816375255585, -0.5069795846939087, -0.3311627507209778, 1.6633621454238892, 0.8951905965805054, 0.6639277935028076, -1.5624101161956787, 1.2253228425979614, -1.3008065223693848, 2.2232470512390137, -0.4967209994792938, -0.02275100164115429, -0.14260943233966827, 0.08814682811498642, -0.6367435455322266, 0.5442110300064087, 0.9115024209022522, -0.8788326978683472, -0.22176700830459595, 0.9068222045898438, 0.7820969223976135, -1.9403094053268433, 0.9962109327316284, 0.46165913343429565, -0.2271798700094223, -0.6440523266792297, -1.8169506788253784, 1.7052470445632935, -1.1179569959640503, 1.4482612609863281, 0.14530183374881744, 0.47348901629447937, -0.5424546599388123, -0.9754688739776611, 0.6045718193054199, -1.0189508199691772, -0.29974421858787537, 1.0918023586273193, 0.5703961849212646, -1.598214864730835], [1.0916308164596558, -1.6381679773330688, -0.169958233833313, 0.6869763135910034, -0.7026675939559937, 0.7116084694862366, -0.538727343082428, -0.3696196675300598, -0.3914017379283905, 1.0748519897460938, -0.6224707961082458, -0.5082393884658813, 0.021772701293230057, 2.332869052886963, 2.26131272315979, -0.09589886665344238, 1.3841142654418945, 0.17838843166828156, -0.2230513095855713, 1.8143359422683716, -0.23586414754390717, -2.113065719604492, -0.2165055274963379, -0.3446047008037567, -0.04078386723995209, -0.5981741547584534, 0.12500275671482086, -0.029762614518404007, 0.775300145149231, 1.197148323059082, -1.349913239479065, 0.6169272661209106, -0.8831488490104675, -0.8173583149909973, 1.1340371370315552, -0.1409696787595749, -0.9345551133155823, -0.0011723022907972336, 0.22897139191627502, -1.329994797706604, 0.10877598077058792, -0.2687074840068817, 1.1473041772842407, 0.43496620655059814, 0.6962064504623413, 0.9987988471984863, -0.28161191940307617, 2.174616813659668, 2.2428996562957764, -0.21643735468387604, 0.6613761186599731, 0.9177119135856628, 1.2751102447509766, -1.1249545812606812, -0.3235982358455658, -0.09447291493415833, 0.664076030254364, 0.7480437755584717, -0.7796148061752319, 0.7532975673675537, -0.48086991906166077, -0.8079856038093567, -1.5935609340667725, -1.4279084205627441, 1.4698244333267212, 1.3489453792572021, -0.10742652416229248, -1.3130922317504883, -1.0493676662445068, -0.8012628555297852, 1.03914213180542, 1.9633808135986328, -1.3901764154434204, 1.696142554283142, -1.9932624101638794, -0.5109761953353882, -0.06532174348831177, 1.659759759902954, 0.8389346599578857, 1.5768682956695557, -1.2958223819732666, 0.2176954746246338, 2.2278006076812744, 0.8841326236724854, 2.22240948677063, -0.9270198941230774, 0.1565299928188324, 0.42642566561698914, -1.2650933265686035, 1.6537591218948364, -0.010908938944339752, 0.2082534283399582, -1.5071784257888794, -0.25663936138153076, 1.0145759582519531, 0.7740088701248169, -1.2221877574920654, 0.8948034644126892, 0.4328605532646179, -0.010726094245910645, 0.17661726474761963, -0.419329434633255, -0.4636453092098236, 0.6250085234642029, -0.11520275473594666, -0.8871819972991943, -0.7037836313247681, 0.2456938475370407, 0.4042391777038574, -0.3466941714286804, 0.3024388551712036, 0.07709647715091705, -1.2012033462524414, 0.9970953464508057, -0.9358325004577637, 1.2427679300308228, 1.2671475410461426, -0.2945513129234314, -0.5362306833267212, -0.2819395959377289, 0.6663253903388977, -1.8457621335983276, 0.8052084445953369, -0.8203404545783997, 0.4977955222129822, -1.725554347038269, 0.15966685116291046, -0.1388917714357376], [-0.05039847642183304, -0.3143625259399414, -1.4973818063735962, 0.11274950951337814, 0.3628145158290863, -1.3545387983322144, -0.9856789112091064, -0.01683737151324749, 0.6154146194458008, -1.2457174062728882, -0.2605387568473816, -0.3244799077510834, 0.3024706244468689, -0.17065005004405975, 1.6843087673187256, -1.595523715019226, 0.7856277823448181, 0.7355853915214539, 0.2276192456483841, -1.0321918725967407, 0.727849006652832, -1.2883409261703491, 0.6398186087608337, 1.3265256881713867, 0.3853282928466797, -1.2217005491256714, -0.05121060088276863, 1.8150814771652222, -2.620974540710449, 0.06715047359466553, 0.4523511230945587, 1.3743969202041626, -1.0041635036468506, -0.6666487455368042, 1.1231366395950317, -1.805788278579712, 1.3374453783035278, -0.6839849352836609, -0.08977366983890533, -0.6888865232467651, -0.9923465847969055, -1.5453212261199951, 0.07273631542921066, 0.9639766216278076, 0.7094314694404602, -0.022748364135622978, -1.79636812210083, 1.969823956489563, 0.5653128027915955, -1.2959727048873901, 1.0348807573318481, 1.1505354642868042, 0.47555503249168396, 0.18311497569084167, -0.8059744238853455, 1.3224244117736816, 1.3795363903045654, 0.7332776188850403, 0.6102445125579834, -0.5546830892562866, 0.002854633843526244, 1.1082841157913208, -0.4905339181423187, -1.0797455310821533, 0.2849859893321991, 0.4182202219963074, -0.21261145174503326, 1.1290920972824097, -0.8377382755279541, -0.356872022151947, 0.47799935936927795, 0.9043005108833313, 0.8625947833061218, 1.946594476699829, -1.5187474489212036, -0.9735564589500427, -0.7697905898094177, 0.4530424177646637, 0.5605730414390564, -1.1364742517471313, 1.4188193082809448, 2.3597540855407715, 0.6468399167060852, 0.4114994406700134, 0.13556496798992157, -0.005411085672676563, 0.7578100562095642, -2.4528207778930664, 0.2034560889005661, -0.7757339477539062, -0.0729367583990097, 0.6306728720664978, 0.9477847218513489, -0.7511662244796753, -1.5901992321014404, 2.1973977088928223, -2.402527093887329, -1.4019207954406738, -0.43735846877098083, 0.8116924166679382, -1.6890507936477661, -0.22470708191394806, 0.07044456899166107, 1.3310790061950684, 0.5127508044242859, -1.611053228378296, -0.6400274634361267, 0.2335021197795868, 1.0026941299438477, 2.9752140045166016, 1.0211808681488037, 1.0296798944473267, 1.077511191368103, -1.173815369606018, -0.20435132086277008, -0.32730403542518616, 0.5980780720710754, 0.878990113735199, 0.19846555590629578, -0.7558797597885132, 0.9327235221862793, -0.679492712020874, 0.418962299823761, -0.4472767412662506, 1.1357619762420654, 0.9633265733718872, 0.6285474896430969, 0.5015641450881958], [1.0130776166915894, -0.7864014506340027, 0.5113205909729004, 1.389291763305664, -0.7707883715629578, -0.561184287071228, 0.1267482042312622, 1.8030036687850952, 1.4434617757797241, 0.6932404637336731, 1.0357518196105957, 0.23365771770477295, -2.4739491939544678, 0.8066311478614807, -1.428288459777832, 0.353059321641922, -0.5826242566108704, 0.12335943430662155, -0.3074113130569458, -0.5215949416160583, 0.9225099086761475, 1.724492073059082, 0.9102739691734314, 0.3337315022945404, 1.1886636018753052, 1.126879334449768, -1.4131865501403809, 0.3066501319408417, 1.4518427848815918, -1.7547643184661865, 0.8012366890907288, 1.4022866487503052, 0.022460052743554115, -1.2596111297607422, -1.1998603343963623, 1.0560745000839233, -0.20011547207832336, 0.1404639631509781, 0.8407453894615173, 0.26884210109710693, -0.2373456507921219, -0.9228209853172302, 1.0645990371704102, 0.18532906472682953, 0.01983194798231125, -1.465640664100647, 1.185648798942566, 0.032838840037584305, 0.8221725225448608, -0.7181583046913147, 2.0746355056762695, -1.3908690214157104, -0.6656644940376282, -0.5371475219726562, -0.9049251079559326, -0.2215675413608551, -1.5490542650222778, 0.7170756459236145, 0.41560110449790955, 0.9357171654701233, -0.00031023818883113563, -0.7868099808692932, -0.7753104567527771, -1.2495068311691284, -1.5949277877807617, -0.4845733940601349, -0.6498680114746094, 2.0858097076416016, 0.573018491268158, -1.7294713258743286, -0.6249144673347473, -0.2705182433128357, -0.526275098323822, 0.648319661617279, 1.1553822755813599, 0.3693240284919739, -1.1547964811325073, -0.019475262612104416, -0.26862311363220215, 0.7424633502960205, 0.9194284677505493, 0.5152742862701416, 0.0511002391576767, -0.5796212553977966, 1.7153724431991577, 1.6784393787384033, 0.3906487226486206, -0.07715217024087906, -1.219067096710205, -0.3239510953426361, 0.21840673685073853, -0.06106535345315933, 0.04790987819433212, 0.20991484820842743, -0.9219363331794739, -0.40526801347732544, -0.23353207111358643, -0.5438661575317383, -1.0416990518569946, 0.9533186554908752, -1.0006519556045532, 0.1631958931684494, 1.7788619995117188, 1.1876795291900635, -2.528144598007202, 1.3516124486923218, -0.29943227767944336, -0.4918513298034668, 0.1432773470878601, -0.23774003982543945, 1.1695481538772583, 0.7144498825073242, -0.7689449191093445, 0.9166256189346313, 1.3289536237716675, 0.2779086232185364, -1.1524094343185425, 1.0568881034851074, 0.75933438539505, 0.7558808922767639, 0.3179934620857239, 1.7211037874221802, 0.42881399393081665, -0.089866504073143, 0.13551335036754608, 1.0256143808364868, 0.8633447289466858, 1.1018683910369873], [0.4748218357563019, -0.8180170655250549, 0.5979294776916504, 0.9146180748939514, -0.6431229114532471, -0.168821319937706, 0.33931905031204224, -0.9168443083763123, 0.33068087697029114, -0.1960698962211609, -0.3872237205505371, 0.40290528535842896, 0.9002314805984497, 0.4675845205783844, 1.7673759460449219, -0.6529120206832886, -0.988325297832489, -0.3507428467273712, 0.8222880959510803, 1.0356122255325317, 1.7935235500335693, -1.060024619102478, 0.48283228278160095, -0.24154913425445557, -0.7426527738571167, 1.7612380981445312, -0.45662692189216614, 2.3955202102661133, 0.23757560551166534, -0.5226697325706482, -0.8093900084495544, 1.1210464239120483, 0.8475357890129089, -1.0341200828552246, 0.8122954964637756, 0.8686690926551819, -0.5146943926811218, -0.9869028329849243, 0.28732770681381226, 0.3174700438976288, 0.41740214824676514, 1.8365432024002075, 1.2015846967697144, 0.3750177025794983, -1.064389705657959, -0.6704902052879333, -0.4179832637310028, -0.15310660004615784, -1.299030065536499, 0.49727174639701843, 0.755536675453186, 0.11785750836133957, -1.4399547576904297, 0.05397282540798187, -1.1329169273376465, 0.9951728582382202, 1.4366739988327026, -0.33887749910354614, -0.3974493443965912, -0.8086950182914734, 1.2956280708312988, -0.9364774823188782, -0.08156483620405197, -0.298302561044693, 1.4658730030059814, 0.6841403841972351, 0.8081991076469421, 1.4454658031463623, 0.6897465586662292, 0.0421527735888958, -0.2614865303039551, -1.2173173427581787, 0.034591224044561386, -0.7963642477989197, -0.6973084211349487, -1.016822338104248, -0.9049167037010193, 0.34689369797706604, 0.25200095772743225, 1.7426091432571411, 1.0333919525146484, -0.09239412099123001, -0.15976227819919586, -1.2510206699371338, 1.1721718311309814, -0.04225703701376915, 0.8157657980918884, 0.18546494841575623, 0.8013362288475037, 1.1562755107879639, -0.8389776349067688, 0.4539191424846649, -1.923758625984192, 0.8500393629074097, -0.21949301660060883, -0.25238773226737976, -1.2788810729980469, 0.03677183762192726, 0.9101310968399048, 0.968345046043396, -2.332314968109131, 1.4737366437911987, -0.17554248869419098, 0.1480211317539215, -0.11878664046525955, -0.997326672077179, -0.7154873013496399, 0.050879500806331635, -0.9209209680557251, 0.8688066601753235, 2.1678860187530518, 1.4799433946609497, 0.8779578804969788, 1.0175641775131226, 1.0862327814102173, 0.21445614099502563, 1.1025618314743042, -0.04827028512954712, -0.7947803139686584, 1.2711788415908813, 0.20445001125335693, -1.7257035970687866, -0.21604380011558533, 1.19480562210083, 1.1608319282531738, 0.4979967772960663, 0.9153872728347778, -0.3081999719142914], [-0.27306050062179565, -2.559305429458618, 0.6971169710159302, 1.432759165763855, -0.8880359530448914, 1.8030766248703003, -1.346561074256897, -1.455372929573059, 0.7672550678253174, -0.4516466557979584, -0.011441764421761036, 0.4570502042770386, 0.061487894505262375, -0.5523189902305603, 0.17562462389469147, 0.6012321710586548, 0.7844652533531189, 0.478372186422348, -0.7064924240112305, -1.6200745105743408, -1.2783052921295166, 1.0759670734405518, 0.7081963419914246, 1.0004229545593262, 0.016963375732302666, -0.10074379295110703, -1.2423090934753418, 0.6182340383529663, -0.11281854659318924, -1.2984507083892822, -0.5592150092124939, -0.056987933814525604, -1.6423677206039429, -1.8327646255493164, 0.04800441116094589, 0.32715728878974915, -0.24974779784679413, -0.1605021059513092, 0.14502201974391937, 2.1025984287261963, 0.45362022519111633, -0.7017590403556824, -1.495262861251831, -0.1382075697183609, 0.12193349003791809, 1.1802492141723633, -0.7354224324226379, 0.41786032915115356, 1.402494192123413, 0.22753137350082397, -0.19004619121551514, -0.5021227598190308, 0.5309950709342957, 0.9900112748146057, 0.28954702615737915, 0.8701871633529663, 0.1389520913362503, -1.9429477453231812, -1.1354435682296753, -0.9450462460517883, 0.21625031530857086, 0.523011326789856, 1.495356798171997, 0.3167143166065216, -1.5552459955215454, 0.37152963876724243, 0.5835583806037903, -1.1640328168869019, 0.12880055606365204, 1.34133780002594, -1.4846857786178589, 0.6313617825508118, -0.4361873269081116, -1.1870486736297607, 0.3299020826816559, 0.5938827395439148, -0.3616688847541809, 0.4064297676086426, -0.881274402141571, 0.31726253032684326, -1.6430355310440063, -0.06851158291101456, 1.0800492763519287, 0.3661768138408661, -0.5386753082275391, -0.6093199849128723, 1.1045547723770142, 0.9755920171737671, 2.3576765060424805, -0.4837775230407715, 1.246504545211792, 2.022608995437622, -0.5297451019287109, -0.560862123966217, 0.05412410572171211, 0.9447271227836609, 0.4030102789402008, 0.34333357214927673, -1.5933449268341064, 0.08404704928398132, -0.6520775556564331, -0.06339067220687866, -1.9282915592193604, 0.9893285632133484, 0.36571434140205383, 0.07906966656446457, -0.4408225119113922, 0.991121768951416, 1.0876221656799316, 0.7021886706352234, -0.6383882761001587, 0.11275674402713776, 0.3701398968696594, 0.8828790187835693, -0.6583892703056335, 0.3447711765766144, 1.9253301620483398, -1.2689317464828491, 0.6535844802856445, 0.8903228044509888, 0.38533079624176025, 0.2143777757883072, -0.5731050372123718, -0.668518602848053, 0.43514731526374817, -0.5212464332580566, -1.0985103845596313, -0.22215218842029572], [0.6906363368034363, 1.4635878801345825, 0.7326852083206177, 0.7735317349433899, 0.4666048586368561, 1.7017161846160889, 0.906491756439209, -1.4019068479537964, -0.9269092679023743, 0.20358368754386902, 0.5639017820358276, 0.9335504770278931, 0.5430047512054443, -0.5564557313919067, 0.04504978284239769, -1.3141720294952393, 0.6373755931854248, 2.1693062782287598, 0.9257735013961792, -1.104500412940979, 1.3137744665145874, 0.42103588581085205, -1.4755933284759521, 0.5320123434066772, 0.4869793653488159, 2.348322868347168, 2.6096668243408203, 0.40473148226737976, -2.0842106342315674, 0.12729448080062866, 0.6965944766998291, 0.5658770799636841, -1.470504879951477, 0.9425297379493713, -0.10079779475927353, -0.4972164034843445, 0.38620737195014954, -2.239680290222168, -0.9162468910217285, 0.3950723707675934, 1.135541319847107, 0.6521949768066406, 0.02340070530772209, -1.3457432985305786, 0.5112170577049255, 1.047935128211975, -1.5650578737258911, 0.9901357889175415, -0.057958878576755524, 1.2035082578659058, -0.567807137966156, 0.33871719241142273, 0.4721475839614868, -0.5461891889572144, 1.1388404369354248, -1.1266005039215088, -0.06267300248146057, 0.09252134710550308, 1.0631194114685059, -0.40140727162361145, 0.12254177778959274, 0.7005277872085571, -1.9440971612930298, 0.9830511808395386, -0.12758386135101318, 0.4367816746234894, -0.2996915578842163, -0.8027929663658142, 1.1333142518997192, 0.6400079727172852, -1.4381153583526611, 0.6652176380157471, 0.6066632866859436, 0.5807397961616516, 0.03612314164638519, -0.34685760736465454, -1.1460661888122559, 1.484409213066101, 0.752764880657196, 1.6804139614105225, -0.8802592158317566, 1.3924520015716553, 0.35788872838020325, 1.2309074401855469, 0.14889761805534363, 2.1025257110595703, 1.0703297853469849, -0.12379653006792068, 0.02341112308204174, -2.122095823287964, 0.6389555931091309, -0.11934138834476471, 0.1175151988863945, -0.361572265625, 2.1560218334198, -1.0966366529464722, 1.063509225845337, -0.4491914212703705, 1.3123624324798584, 0.6259152889251709, -2.1734085083007812, 1.5709902048110962, -0.0066031343303620815, -0.8897782564163208, -0.1516292542219162, -2.0641674995422363, 0.2685735523700714, -1.726936936378479, 0.0020473951008170843, -0.059816665947437286, 1.072493314743042, 0.8782391548156738, -0.23239921033382416, -1.6020323038101196, 0.36438941955566406, 0.8604729175567627, 1.2069002389907837, 0.22028294205665588, 0.01876685954630375, 1.359450101852417, 1.4230196475982666, -1.0533329248428345, -0.4990091323852539, 0.42845332622528076, -0.528191328048706, 0.5276411771774292, 0.670991837978363, 1.6817303895950317], [0.7814851999282837, 0.9579151272773743, -1.743471384048462, -0.43704953789711, 0.9608619213104248, -0.637272834777832, 1.9658417701721191, 1.0255441665649414, -0.8826216459274292, 0.6319682002067566, -0.4504416882991791, 1.7251238822937012, 1.6775269508361816, -1.5390211343765259, -1.0162569284439087, 1.4403178691864014, -0.6944465637207031, 0.7555939555168152, -0.6173428893089294, -0.6679045557975769, 1.7539271116256714, 0.9665641784667969, 0.997455358505249, -0.16344887018203735, 2.1349713802337646, 1.1031526327133179, -0.43764162063598633, 0.44647690653800964, 0.3190404772758484, 0.5822596549987793, -1.522853136062622, 0.9817051291465759, -1.1793211698532104, -0.8027621507644653, -1.8279386758804321, 0.49697059392929077, 0.7198778986930847, 2.9195823669433594, -1.2283297777175903, 1.1129014492034912, 0.13372644782066345, 0.3796927332878113, 0.7296591401100159, 0.3458618223667145, 0.5462231636047363, -0.06324509531259537, -1.2534970045089722, 0.40542709827423096, 0.8428739905357361, 0.5600132346153259, 0.7788794636726379, -0.05090494081377983, -1.4910550117492676, -0.4657559394836426, -0.6269223690032959, 1.2319520711898804, -0.13261863589286804, 0.007807028945535421, 0.2441738247871399, 1.7373111248016357, 0.4981499910354614, 0.049355797469615936, 0.24630531668663025, 0.2612394392490387, -1.459193229675293, 1.0337578058242798, 1.3265565633773804, -0.12539198994636536, 1.638235330581665, -0.02192213386297226, 0.02105989120900631, 0.013723615556955338, 1.2303987741470337, 1.626407504081726, 0.3215619623661041, 0.7996739745140076, -0.47732922434806824, 0.18122407793998718, 1.8127315044403076, 0.6681591272354126, -0.44620105624198914, 0.5967647433280945, -0.17498897016048431, -0.3190736174583435, 0.1955123096704483, 0.1667749285697937, 0.9808434844017029, 0.40594708919525146, 0.06790940463542938, -0.32606953382492065, -0.543584406375885, -0.5475950837135315, 1.427082896232605, -1.6800837516784668, -0.03129152953624725, -0.9236361384391785, -3.4787614345550537, -1.8122004270553589, 0.5853309631347656, -0.2200053334236145, -0.24190907180309296, -1.1090137958526611, -0.8353268504142761, 0.42199352383613586, -0.4712119400501251, 0.6556161642074585, 2.7962357997894287, -0.663687527179718, 1.0835589170455933, -0.016624508425593376, 0.8873729109764099, 0.7692455649375916, -0.6674069762229919, -0.540645956993103, 0.17815490067005157, 0.016453152522444725, 0.06019490584731102, -1.7603228092193604, 0.5545745491981506, 1.6985429525375366, -0.4414074420928955, 0.3439847230911255, 1.2124437093734741, 0.9245179295539856, 0.5327108502388, -2.490004301071167, 0.4642161726951599, -1.6767646074295044], [-0.9837774634361267, 0.5387794375419617, 0.7946959137916565, -0.08437465131282806, -1.1679890155792236, 1.230857253074646, 1.5344009399414062, -1.729162335395813, 0.4413378834724426, 1.5745924711227417, 1.3444105386734009, -0.19646891951560974, 0.812088668346405, -0.817263126373291, -0.7460572719573975, 1.725645661354065, -1.1502830982208252, 0.600782036781311, 0.3649385869503021, -0.49836465716362, 0.1654425412416458, 1.5379053354263306, 0.3669579327106476, -0.8026655316352844, -0.5844297409057617, -1.7564353942871094, -0.6044775247573853, -0.6832722425460815, 0.882870078086853, -0.6708675026893616, -1.1247481107711792, -0.2536989450454712, 0.17755702137947083, 0.7564527988433838, 0.9783036708831787, 0.8443271517753601, 1.2194793224334717, -0.8411509394645691, 0.6890304088592529, -0.07106412947177887, 0.9755902290344238, 0.6995198130607605, 0.4628952145576477, -0.09234698116779327, 0.09991611540317535, 0.8851719498634338, 0.5923153758049011, 0.5292711853981018, -0.013728086836636066, -0.32631805539131165, -0.8737662434577942, 0.2606232762336731, -1.1919567584991455, 1.553001880645752, 0.09215854108333588, 0.6416468024253845, -0.48835793137550354, 1.205021619796753, -0.43941566348075867, 1.5175944566726685, -0.2699657380580902, 0.05297054350376129, 0.6618666648864746, -0.19354936480522156, 1.386527419090271, -0.7708734273910522, 1.0435467958450317, 0.8734093308448792, -0.13252165913581848, -0.6086121201515198, -0.6741824746131897, -0.22287042438983917, 0.17706415057182312, -0.2540648877620697, -0.42124301195144653, -0.24515904486179352, -1.2849609851837158, -1.0164247751235962, 0.5892243981361389, 1.3050997257232666, -0.3875740170478821, 0.6486984491348267, -0.737540066242218, 1.2997138500213623, -1.3520054817199707, 1.8719688653945923, -0.7933713793754578, 1.2340822219848633, 0.472206711769104, -0.8291341662406921, -0.3652055859565735, 0.7871817946434021, -1.2533555030822754, -0.17270062863826752, -0.941015899181366, -0.27108192443847656, -0.052363600581884384, 0.8444260358810425, 0.3232574760913849, 0.9911225438117981, 1.2031317949295044, -0.5627318024635315, 0.6060361862182617, -0.021045144647359848, -0.7553588151931763, 0.972697377204895, 0.8727817535400391, -0.31352749466896057, -1.2904613018035889, 0.8199365139007568, 0.2721622884273529, -0.34203776717185974, 1.23099684715271, 1.42641019821167, -0.5114330053329468, -0.46372511982917786, 2.580263614654541, 0.7545417547225952, 1.6130996942520142, -0.22289182245731354, -1.5350912809371948, 0.8864641785621643, 0.04690433293581009, -0.5441312193870544, 0.9218274354934692, 0.501300036907196, -0.8054908514022827, 1.1401705741882324]]], [[[0.7095609903335571, -0.3100678026676178, 0.02548939548432827, -0.344625860452652, -0.14295068383216858, 1.397107481956482, 2.2064664363861084, -0.19431452453136444, 1.6592422723770142, -2.0094351768493652, 0.0024937093257904053, -1.0304322242736816, -0.6677717566490173, -0.35297101736068726, 0.5359212756156921, -1.8288315534591675, 1.6187727451324463, 0.4192913770675659, 0.09636656939983368, -0.9016150236129761, 0.11111829429864883, -1.622718334197998, 0.8482100963592529, -1.6371970176696777, 0.4861307442188263, 0.5023747086524963, -0.34841224551200867, -0.8790308237075806, 0.44404518604278564, -0.9981102347373962, 0.2348157912492752, 0.6981332898139954, -0.14943836629390717, 2.245142698287964, 1.435221791267395, -1.5515785217285156, 0.2518267035484314, 1.5382394790649414, 0.06649217754602432, -1.2013237476348877, 0.3398802578449249, -0.3340010344982147, 0.5053480863571167, 0.2497130036354065, -0.44873300194740295, -0.4223436713218689, 0.03407580032944679, 0.5573023557662964, -0.9365904331207275, -0.42767345905303955, 1.349822998046875, -2.1492419242858887, -0.45919761061668396, -1.2981878519058228, -0.712800145149231, -1.694770336151123, 0.21853893995285034, -0.5448921322822571, 1.4800657033920288, 0.9462641477584839, 0.9267970323562622, 0.4367235004901886, 1.2326197624206543, -0.6682659983634949, 0.0018998711602762341, -1.8276275396347046, 1.2109116315841675, 0.06370221078395844, -1.0291649103164673, 0.38700398802757263, 0.4842725694179535, -1.4250248670578003, 0.48081934452056885, -1.3450285196304321, 1.248300313949585, -1.5581589937210083, 0.4986182749271393, 0.3669968545436859, 1.6318858861923218, 0.8474560976028442, -0.5370607972145081, 0.09815066307783127, -1.6179354190826416, 1.9589084386825562, -0.07900083065032959, -0.06494121253490448, -1.0047377347946167, -0.3266131281852722, -1.119523286819458, 0.08148818463087082, 1.1725186109542847, -0.10605094581842422, 1.3018473386764526, 1.1407610177993774, -1.1686203479766846, -0.7290371656417847, 0.781297504901886, 2.4660701751708984, 1.0747289657592773, 1.9977599382400513, 0.7509638667106628, 1.7418774366378784, 1.8748128414154053, 0.6137615442276001, -1.4679168462753296, 0.9247952699661255, 1.0967832803726196, 0.49485427141189575, 0.9803107976913452, -0.7222751975059509, -0.6398507952690125, 1.5904518365859985, 1.4342155456542969, 1.4536724090576172, -0.3162442743778229, 0.41459986567497253, 0.1352739930152893, 0.9057160019874573, 0.3303523361682892, 1.6239436864852905, -0.7629119157791138, 0.5608347654342651, -0.7273001074790955, -0.594033420085907, -0.04574596509337425, 0.2700200378894806, 1.7508149147033691, -0.5676645636558533], [-0.21157503128051758, 0.7246447801589966, -0.03525139018893242, -0.7885043621063232, 0.3729645907878876, -0.14784783124923706, 1.837827444076538, 1.4747122526168823, 0.4959907829761505, -0.4660688638687134, -1.855728030204773, 0.5144109725952148, -0.14419357478618622, -0.04151706397533417, -0.2597874402999878, -0.5183874368667603, -0.9656013250350952, 1.0386286973953247, 0.4242595136165619, -1.3642185926437378, 0.10514860600233078, 0.9913772940635681, -0.7554590106010437, -0.3643505573272705, 0.8229019641876221, -0.050379056483507156, 0.6879026889801025, -0.8663199543952942, -0.0876118615269661, -1.9639842510223389, -2.0757360458374023, -0.427918404340744, 0.5530077815055847, -1.7410132884979248, -1.23101007938385, 1.555031418800354, 1.3203879594802856, 1.2101048231124878, -0.4626675546169281, -0.6466664671897888, 0.9608833193778992, -0.1272413283586502, 0.4526243209838867, -0.5283567309379578, -1.3577475547790527, -2.1231791973114014, -0.23298878967761993, 0.32672345638275146, -1.4596855640411377, 0.10143397003412247, -0.8705399036407471, 0.11117946356534958, -0.0684567466378212, -1.1210459470748901, -0.0033523780293762684, -0.3428342938423157, -0.45549964904785156, -0.8268100619316101, -0.6020504832267761, 0.6745844483375549, 0.6602571606636047, -1.1392806768417358, -0.0011795469326898456, 0.1198834776878357, -0.813653826713562, 0.40010806918144226, 0.5576699376106262, 1.8578742742538452, -1.575183629989624, 0.53602534532547, -0.7010542750358582, 0.3627058267593384, 2.1753244400024414, 0.39457055926322937, -1.2124600410461426, -0.541968047618866, 0.6899423599243164, -1.6103205680847168, -0.2075660526752472, -2.2712385654449463, 1.098285436630249, 0.37535983324050903, 1.5884414911270142, -0.6400452256202698, -0.8882083296775818, 0.6009417772293091, 0.5782752633094788, -1.2164400815963745, 0.17207182943820953, -0.9471033215522766, -0.9832548499107361, 0.8024178147315979, 0.733158528804779, -1.0191618204116821, 0.3007884621620178, 0.7971322536468506, -1.0189387798309326, 1.3021186590194702, -0.6081159114837646, -1.7710273265838623, -0.8409290313720703, -0.1810731440782547, -1.6557427644729614, 1.7963018417358398, 0.4095107316970825, -1.5046272277832031, 0.9581727385520935, 1.087716817855835, -0.10969409346580505, 0.6800172328948975, 0.9820102453231812, 1.0117990970611572, -1.7114540338516235, -0.08422813564538956, -1.2136716842651367, 0.020055681467056274, -1.2837412357330322, -1.2897857427597046, 0.7639549374580383, -0.20584669709205627, -1.0981426239013672, 1.3961687088012695, -0.6708041429519653, 2.2066831588745117, -1.18607759475708, -0.9073439836502075, -1.040767788887024, 0.7883951663970947], [-2.560645818710327, -1.8768095970153809, -0.574080228805542, 0.8041850328445435, 0.7010823488235474, -0.7649420499801636, -1.0302373170852661, -0.06440266221761703, 1.0349727869033813, 0.048932548612356186, 0.5826547145843506, -1.427951693534851, -0.48388272523880005, 0.9298500418663025, 0.46860218048095703, 0.22450268268585205, 0.054961416870355606, 0.8184080123901367, -0.45240432024002075, -0.6253446340560913, 0.06539148092269897, -0.90373295545578, -1.7590752840042114, -0.3550858497619629, -1.3064236640930176, -0.006910952273756266, 1.0065730810165405, -0.7990081906318665, -0.15061314404010773, -0.36097443103790283, 0.4712299406528473, -1.2728193998336792, 0.4445502460002899, -1.6296442747116089, 0.027220968157052994, 0.7680389285087585, -2.3335745334625244, 1.9680438041687012, -1.92465341091156, 0.9386591911315918, 1.8637721538543701, -1.038041591644287, -0.7822692394256592, 0.9023188352584839, -0.2968488335609436, -2.3772389888763428, 1.2041311264038086, 1.2213611602783203, 0.5823461413383484, -0.593573808670044, -1.1877903938293457, 0.4425652027130127, 0.9154548645019531, -0.8670209050178528, 0.5874084234237671, 0.05793910101056099, 0.7779324054718018, -1.1463041305541992, -0.5186182856559753, 0.44887253642082214, -0.15160097181797028, 0.25976982712745667, -1.8611069917678833, -1.2678062915802002, -0.6916516423225403, 0.3867485821247101, 0.15536761283874512, 1.368916392326355, 1.9059414863586426, -0.35405421257019043, 0.3737228810787201, 2.1395132541656494, 0.353371262550354, 0.9014610052108765, 0.8159493803977966, 0.020991511642932892, 1.2933018207550049, -1.8732118606567383, 1.5871700048446655, -1.7188172340393066, -0.13680051267147064, 0.230804905295372, -1.0159834623336792, -2.0100324153900146, -0.6781928539276123, -0.5371352434158325, 0.7657476663589478, -0.4856128692626953, 0.7585217356681824, -0.4163074493408203, -1.2853963375091553, -0.04106632620096207, -0.720342218875885, -1.2765923738479614, 0.7949714064598083, -0.9223031401634216, 0.15320327877998352, -1.3164052963256836, 0.918271005153656, -0.7048671245574951, -0.5689424872398376, -1.0484678745269775, -1.2529999017715454, -1.3850051164627075, 0.5066596865653992, -2.0092947483062744, 1.0117496252059937, -2.8092851638793945, -2.439206600189209, -1.6511142253875732, -0.5652646422386169, -0.39075639843940735, -1.6014362573623657, 1.3407281637191772, -2.1955924034118652, -0.7229527235031128, 0.7329875230789185, -0.7545042037963867, 0.06975012272596359, -1.382433533668518, -0.06659014523029327, 0.1116010844707489, 0.9964025616645813, 0.32936620712280273, 0.26373106241226196, 0.377518892288208, -2.4138829708099365, 0.144968643784523], [-0.21996240317821503, 0.4839319586753845, -0.8191623091697693, 1.2420226335525513, 0.6238437294960022, 0.8288214206695557, 1.6779192686080933, 1.2034848928451538, -0.13617496192455292, -1.0134872198104858, 1.2698549032211304, 2.621023654937744, -0.9106560349464417, -0.9471918344497681, 0.13883577287197113, 0.21991485357284546, -0.18967905640602112, 0.6241891980171204, 0.3304939866065979, 0.9150122404098511, 0.6411035656929016, -2.5392000675201416, -0.9650190472602844, -2.188643455505371, 1.576101541519165, -0.8217648267745972, -0.3141951858997345, -2.4085376262664795, 0.36830389499664307, -0.7347213625907898, 1.2607735395431519, -0.980608344078064, -1.3320729732513428, 0.8660256266593933, -0.6353670954704285, -0.35042300820350647, -1.126863956451416, -0.8561150431632996, 0.031273484230041504, -0.5716098546981812, 1.16032874584198, -0.816666841506958, 0.34876924753189087, 0.32063573598861694, 0.19972053170204163, 1.0023976564407349, -0.6187996864318848, -0.03878433629870415, 0.20213907957077026, -1.3839746713638306, 0.5534287095069885, 1.1128016710281372, -0.6356409788131714, -1.0966320037841797, 0.04468469321727753, 0.008940390311181545, -0.9729533195495605, -0.6543021202087402, 0.49991676211357117, -0.04948442056775093, -0.9233524799346924, 1.386580228805542, 1.168082356452942, -0.29977768659591675, 1.038026213645935, -0.9035940766334534, -0.4080961346626282, -1.156294822692871, 0.40972164273262024, 0.1985684037208557, 0.484274297952652, -0.7322864532470703, 1.2994893789291382, -1.3924143314361572, -0.7223258018493652, 0.26162198185920715, 0.526932954788208, -0.9580810070037842, 0.6020072102546692, 0.27208444476127625, -1.1973515748977661, -1.025638461112976, -0.6146632432937622, 0.06477976590394974, 0.6115216016769409, -1.0904009342193604, 0.1029900535941124, 0.8447807431221008, -0.09267120808362961, 0.42731404304504395, 0.010670226998627186, 0.040202800184488297, -0.05231394246220589, -0.5626668334007263, -0.3708212971687317, 1.698833703994751, -0.9114472270011902, -0.8022010922431946, -0.01848961040377617, -1.2367236614227295, -0.11161531507968903, 0.45701146125793457, 0.5624948740005493, 0.652077317237854, -0.9962493181228638, 0.14808660745620728, -0.5354242324829102, 1.9950579404830933, 1.8804020881652832, -0.6109813451766968, 0.4887455403804779, -0.25936830043792725, -1.254624605178833, -0.029321104288101196, -0.22034962475299835, 1.3429388999938965, -0.9319255352020264, -0.6541143655776978, 0.3969924747943878, 0.547258198261261, -1.441084623336792, -0.8608658313751221, 1.8103290796279907, 1.3354572057724, -0.4384866952896118, -1.7143033742904663, 0.7639206051826477, -0.1946375072002411], [0.33515071868896484, 1.4950443506240845, -0.620689332485199, -0.9006885290145874, 1.6631501913070679, -0.937069833278656, -1.392927646636963, 0.00784364901483059, 0.8619669079780579, -0.8753177523612976, 0.1462676227092743, 0.28703415393829346, -0.6794686317443848, -0.6103346347808838, 0.8025175929069519, -2.14689040184021, 1.2768617868423462, -0.7226853370666504, -0.7860518097877502, 0.5923957228660583, -0.7835215926170349, 0.772018551826477, 0.43433836102485657, -1.4339348077774048, -1.1396194696426392, -0.5010199546813965, 2.6410067081451416, -1.6799012422561646, -0.0474918968975544, -0.7207555174827576, 0.5463778376579285, 0.874193549156189, 0.7408170700073242, 0.08447924256324768, -0.8937423229217529, -1.6956130266189575, -0.9756563305854797, 0.1505577564239502, 1.1484531164169312, 0.9963087439537048, 0.2769124507904053, 0.10031091421842575, 0.21837402880191803, -0.29241666197776794, -0.3438027501106262, 1.0062364339828491, -0.986008882522583, -0.8332789540290833, 0.2920582890510559, 2.441265106201172, 0.39518657326698303, -0.022613724693655968, 1.811614751815796, -0.17164306342601776, 0.1933802217245102, -1.5360536575317383, -0.5098064541816711, -1.4382604360580444, 1.945071816444397, -1.5070730447769165, 1.0124856233596802, 1.2648416757583618, 1.3388937711715698, -0.14768506586551666, -0.05786745995283127, -1.8237935304641724, 0.6366629600524902, 2.0504000186920166, 0.5829384922981262, -0.8122042417526245, 1.0202052593231201, 0.25026270747184753, -1.4208660125732422, 0.3543013632297516, 0.44479572772979736, -1.101549744606018, -0.0037625099066644907, -0.09027904272079468, -0.7501217722892761, 0.7421330213546753, 0.0956350713968277, 0.7576484084129333, -1.191959023475647, 0.6565957069396973, -0.5460116267204285, -1.485467553138733, -0.8257168531417847, 1.0372403860092163, -0.16714884340763092, 0.14868998527526855, 0.5354427695274353, 0.007928377017378807, 0.5136297941207886, -0.4652785658836365, 0.12942476570606232, -0.8225241303443909, -0.47721585631370544, 1.0772364139556885, -1.0602986812591553, 0.08271081745624542, -1.1735447645187378, 0.2985077500343323, -0.7898061871528625, -1.2103629112243652, -0.037252604961395264, -0.8441043496131897, 0.47526124119758606, 0.5895938873291016, -0.5516898036003113, 0.4709448516368866, -0.544095516204834, -0.11308562010526657, 0.30079150199890137, -1.0729014873504639, -0.4124387800693512, -0.30763742327690125, 0.4927518367767334, 0.19041231274604797, 0.10283038765192032, 0.4358896315097809, 1.1554690599441528, 1.3281357288360596, -0.5268080830574036, -1.5642919540405273, -2.312021493911743, -0.08481103181838989, 1.4098994731903076, -0.18351712822914124], [-0.11819427460432053, 0.31429561972618103, 0.30525410175323486, -0.8893412351608276, -0.619070291519165, 1.6407321691513062, 1.6311246156692505, 1.051984190940857, -1.0741655826568604, -1.0490964651107788, -0.5747627019882202, 1.259600281715393, 1.303827166557312, -0.4004456102848053, 0.09098855406045914, 0.6235876679420471, 0.4181317389011383, -1.0472462177276611, 0.3646472692489624, -1.2847157716751099, -0.21674282848834991, -0.46956807374954224, 1.149760127067566, -0.7609553337097168, 0.49898287653923035, 0.5417127013206482, -0.07964634895324707, -0.42111876606941223, -0.4609795808792114, 0.9502678513526917, -1.3414623737335205, 1.5875097513198853, 0.39837926626205444, 0.8312460780143738, 0.17849642038345337, -0.4652954638004303, -0.8002205491065979, 0.7941730618476868, -1.5436550378799438, 0.34768182039260864, -0.6622060537338257, -0.8051729202270508, 0.09601855278015137, -0.5313941836357117, 1.3796882629394531, 0.6739329099655151, -0.10585134476423264, -1.5543887615203857, 0.06280197948217392, 0.9058328866958618, -1.0079249143600464, 0.8042190670967102, 0.0321565717458725, -0.2274010181427002, 0.4525705873966217, -0.6846917271614075, 0.7652007341384888, -1.1797747611999512, -0.3315947651863098, -1.4867446422576904, -1.051770806312561, -0.9795367121696472, 0.3400909900665283, -0.7492274641990662, -0.6305947303771973, 0.7404892444610596, 0.9832180738449097, 0.12095239758491516, -0.8468499183654785, -1.6366896629333496, -1.0900311470031738, -0.848784327507019, -0.5456636548042297, 0.6894954442977905, -0.1476617157459259, 0.039985980838537216, -1.08001708984375, -1.0096495151519775, -0.04000215232372284, 0.2892446517944336, -0.37410154938697815, -1.0697048902511597, -1.8925316333770752, 0.7126491665840149, -1.2702146768569946, 2.257922887802124, 1.90378999710083, -1.768796682357788, 1.2782137393951416, 0.6562036871910095, -0.6215267777442932, 0.31453341245651245, 1.3671319484710693, 2.111868381500244, 1.66818368434906, -0.36840009689331055, -0.6458487510681152, -0.783065676689148, 0.003344569820910692, 0.12050226330757141, -0.02981267124414444, 0.3096218407154083, -3.2172234058380127, -1.8330137729644775, -1.6327251195907593, -0.49126940965652466, -1.118838906288147, -1.3699008226394653, 0.557948112487793, 0.12148021906614304, -1.6864217519760132, 0.8606079816818237, -1.0668981075286865, -0.2641725540161133, 1.128883957862854, 0.4404810667037964, 1.4061195850372314, -0.2176552712917328, 0.7146803140640259, 0.505712628364563, -1.86049222946167, 0.17061366140842438, 0.2975027859210968, 0.0329466238617897, -0.6349102258682251, -1.66975998878479, 2.207792043685913, 0.3876461386680603], [0.1440170556306839, -0.07514727860689163, 0.4485699534416199, -0.5766756534576416, -1.0785342454910278, -1.243464708328247, -0.21760155260562897, -0.3123750686645508, -0.35253363847732544, 0.22889076173305511, 0.21391800045967102, 1.0951722860336304, -1.4607490301132202, 0.6049829125404358, 1.292194128036499, 0.0024981764145195484, 0.45138660073280334, -0.04118593782186508, -0.6646583676338196, 0.38711580634117126, -0.13683784008026123, -2.570152521133423, 0.8062478303909302, -0.8914728760719299, -0.7805073261260986, -2.066555976867676, 0.05063126981258392, 0.828193724155426, 0.611775279045105, 1.5871758460998535, -0.20620937645435333, -0.040936242789030075, -0.5196242332458496, -0.6410073637962341, 1.1195380687713623, 1.6198952198028564, -0.2647032141685486, -0.0838615894317627, -1.1994415521621704, -0.5678105354309082, -0.6986167430877686, -1.511724591255188, 0.47777947783470154, 0.740787148475647, -2.3830411434173584, 0.0031430174130946398, -0.02993060275912285, -1.9359846115112305, 1.362626075744629, 1.4486619234085083, 0.22517108917236328, -0.10458873957395554, 0.13389718532562256, 1.2692201137542725, -0.7298815846443176, -0.16301250457763672, 1.2166980504989624, -1.4981074333190918, 0.05561006814241409, 0.8925838470458984, 0.7683516144752502, 1.3518842458724976, 0.1082841157913208, 0.5296574831008911, -0.3083190321922302, -1.1912871599197388, -0.26482558250427246, 0.7994507551193237, -0.03967323526740074, 0.3235136866569519, -1.7408809661865234, 0.6551970839500427, -3.0315065383911133, 0.1537226140499115, 0.3583691120147705, 2.6992204189300537, -0.20573323965072632, 0.7351101636886597, -0.7727293372154236, 0.3892386555671692, 0.007016806397587061, -0.4639246165752411, 1.5124619007110596, 0.2210320234298706, 0.3248857259750366, 0.4540165662765503, -1.0795305967330933, 0.5092954039573669, 0.205744668841362, 0.40337201952934265, -0.28319790959358215, -0.021228203549981117, 0.5007227063179016, 0.45898306369781494, -0.09382389485836029, 1.1092866659164429, 1.5851715803146362, -0.4422692060470581, 1.0704095363616943, -0.38607755303382874, -0.3402906358242035, -1.4268507957458496, 1.231899380683899, 0.3422594964504242, -0.3048693835735321, 0.9608732461929321, -0.12338338792324066, -0.010138672776520252, 0.04297441616654396, -0.0731513574719429, 0.47115975618362427, -1.2174689769744873, 0.3404582142829895, -0.9474352598190308, 0.35861146450042725, 1.7221266031265259, -0.46112096309661865, 0.7742409706115723, -1.4173253774642944, 0.9403384327888489, 1.3593477010726929, -1.0028495788574219, 1.4572001695632935, -0.09826122969388962, -0.5131340622901917, 0.6150616407394409, -0.6769888401031494, -1.6195200681686401], [0.673835039138794, -0.7360211610794067, 1.2776305675506592, -1.1303839683532715, -1.00129234790802, 0.025185130536556244, -0.903891921043396, -1.2450717687606812, 0.6875767707824707, 0.4159949719905853, 1.1921311616897583, -0.5175303220748901, -0.10968701541423798, 0.8728147745132446, 2.198654890060425, 0.6493366360664368, 0.3601679801940918, -0.05922409147024155, 0.03685491904616356, -1.666140079498291, -0.5448912978172302, -0.1201757863163948, -0.24287284910678864, 0.4981124699115753, -0.5717028975486755, -1.4400802850723267, -0.26956290006637573, 1.9840079545974731, -0.5630086660385132, -0.5446567535400391, 0.5359646081924438, 0.725813627243042, -0.5655882954597473, 1.0629581212997437, -0.11990097910165787, -0.6614517569541931, -0.462155282497406, -0.039869584143161774, 0.5401323437690735, -0.3468705415725708, 0.48843199014663696, -0.9218423962593079, 0.04844330623745918, 0.05532817915081978, 1.7883636951446533, -0.007032702211290598, -0.3107101321220398, 0.32345977425575256, 1.192723035812378, -0.3772053122520447, -0.07611696422100067, 1.393358826637268, -1.0326746702194214, 1.0978937149047852, -1.0612696409225464, -1.7222378253936768, -1.0915669202804565, -0.45970386266708374, 0.09798367321491241, -0.6566680669784546, 0.5257764458656311, -1.3465466499328613, 2.372725009918213, -1.5263527631759644, 1.0222539901733398, -1.636419653892517, 0.36644625663757324, -1.1810917854309082, -0.2514619529247284, 0.8007935881614685, -0.05345752835273743, 0.682131290435791, 0.009865084663033485, 0.5550448298454285, -0.8461341857910156, 0.3841506838798523, -0.24108034372329712, 1.3125152587890625, 1.1283763647079468, 0.06908675283193588, -0.9728766679763794, -0.8978536128997803, -1.531329870223999, -0.6183552145957947, 0.040089014917612076, -0.7017072439193726, -0.33828091621398926, 1.366507887840271, 0.023706207051873207, -0.4441157579421997, -1.3520818948745728, -1.5257245302200317, -2.0308115482330322, -2.040841579437256, 0.1637442260980606, 0.19102077186107635, 0.5625642538070679, 1.6221369504928589, 0.21194083988666534, 1.227211833000183, -0.037879861891269684, -1.3486602306365967, 0.9541476964950562, -1.3493205308914185, -0.348054438829422, 1.252272129058838, 1.632948637008667, -0.6821085810661316, -0.33921104669570923, 1.323616623878479, -0.21707262098789215, -0.747327446937561, 0.5991040468215942, -1.6530576944351196, -0.961591899394989, 1.2568162679672241, 0.14743837714195251, 0.5441649556159973, -1.7533624172210693, 0.9610776305198669, -0.725421130657196, 0.6035835146903992, 1.9459946155548096, 1.3646959066390991, -1.3433220386505127, 0.47024115920066833, -1.6912320852279663, 0.02954866923391819], [0.5377479791641235, -0.4523489475250244, 0.06736057996749878, -0.461322158575058, -0.1489885151386261, -1.4641886949539185, -1.0388182401657104, 0.7775005102157593, -0.6850377917289734, -1.5142117738723755, 0.008662927895784378, 0.6827517151832581, 0.04431305453181267, -1.4281715154647827, 1.9956859350204468, 0.4064348042011261, 2.2120959758758545, -0.3932693898677826, -0.6735345721244812, 0.13513562083244324, 0.10814517736434937, -1.0715103149414062, 0.13817113637924194, -0.01289253868162632, -1.3309131860733032, 0.533886730670929, -0.9357141852378845, -0.782511830329895, 0.3221982419490814, -0.4922088086605072, -0.29554489254951477, -0.2461017668247223, 0.18155746161937714, -0.4666477143764496, -0.872210681438446, -0.2224833220243454, -0.1206907406449318, 1.1262903213500977, 0.7700014114379883, -0.4610595703125, -1.048429012298584, -0.2821306884288788, -0.6112462878227234, -0.5584007501602173, 0.30711430311203003, 2.093402147293091, 0.02968742698431015, -1.2786763906478882, 1.2448869943618774, -0.2951907813549042, -1.4935907125473022, -1.2610719203948975, 0.5835263133049011, 0.4692610800266266, 0.11542298644781113, 0.1440928727388382, -2.3353004455566406, -1.10813307762146, 0.31498560309410095, -0.6896743178367615, -1.0868942737579346, -0.5582907199859619, 1.7736384868621826, -1.7575868368148804, 0.5566301345825195, 0.5695945024490356, -0.5629624128341675, 0.8397612571716309, -1.1814496517181396, -0.2305588722229004, 0.9737185835838318, -2.0289604663848877, -0.7539339661598206, -0.05429365485906601, 0.5916967988014221, 0.2770633399486542, -0.7066303491592407, 0.27193859219551086, 1.7124158143997192, -0.5071701407432556, -0.8589391112327576, 2.4997050762176514, -0.3068436086177826, -0.17494583129882812, 0.3157326579093933, 1.5531271696090698, 0.45110467076301575, -0.2595561742782593, -1.3660274744033813, 0.1580977439880371, -0.28293612599372864, 0.20173805952072144, 0.1801462322473526, 0.15181668102741241, 0.02070450782775879, -0.9117680191993713, 1.4605224132537842, 1.501613736152649, -0.04989419877529144, 0.023000478744506836, 0.3424730896949768, -0.5557767748832703, 0.09764847904443741, 0.79109787940979, -0.2408832609653473, -0.012057581916451454, -1.0740858316421509, -0.8678178787231445, -2.3986597061157227, -1.8263541460037231, -1.9468470811843872, -0.018481165170669556, 0.8962650895118713, 1.8638285398483276, -0.5847963690757751, 0.014810120686888695, -0.5544499158859253, -1.2552073001861572, 0.7384395599365234, -1.357608675956726, 0.011032507754862309, 1.4410221576690674, 0.6297910809516907, 0.9981434345245361, 0.11631569266319275, 0.9928745627403259, 0.2578444480895996, 0.3511494994163513], [-0.7943165302276611, -0.8535111546516418, 1.2431881427764893, 0.7629818916320801, 0.4414006173610687, -2.321087121963501, 0.9736266136169434, 1.3834751844406128, 0.631807267665863, 0.5336431860923767, -1.0914347171783447, -0.5592145919799805, -1.6989022493362427, 0.04206427186727524, 0.7192460298538208, 1.3349359035491943, -2.3064517974853516, 0.17763233184814453, -1.0517529249191284, -0.5528113842010498, -1.2306472063064575, 0.29713961482048035, 0.7351592779159546, -0.822704553604126, -0.5186630487442017, 1.0537060499191284, -0.5849130153656006, -0.4361293315887451, -1.7074134349822998, 0.0011472493642941117, -0.5294129252433777, 0.11738108843564987, 0.7877498269081116, -0.7257717847824097, 0.14229343831539154, -0.6154451370239258, 0.6798392534255981, -0.5962422490119934, 1.536906123161316, 1.216591477394104, 1.1443341970443726, 0.5754926204681396, 1.4850856065750122, 0.3978905975818634, -0.18917493522167206, 1.2984156608581543, 1.7328441143035889, 0.16008156538009644, 0.23412233591079712, -0.22822372615337372, 1.6467814445495605, -1.2013990879058838, 1.1653242111206055, -0.8526560664176941, -0.30452761054039, -1.093827724456787, 1.258953332901001, 0.30909043550491333, -1.5007375478744507, -0.31701380014419556, 0.608463704586029, 1.4475725889205933, 0.35045287013053894, 0.04532020911574364, 0.36995160579681396, -1.1489022970199585, -1.4738047122955322, 0.9169731736183167, -0.3715304732322693, 0.022362850606441498, -0.14883999526500702, 0.3059680461883545, -1.7386314868927002, 0.008989084511995316, 0.3474162220954895, 1.2928721904754639, 0.5185934901237488, 1.725831389427185, -0.5434185862541199, 0.6367554664611816, 0.6126599311828613, 0.7959452867507935, 0.7926507592201233, 0.4713563323020935, 0.052872639149427414, 0.7605148553848267, 0.3046441078186035, 0.13602425158023834, 0.600701093673706, -0.48630291223526, 1.7773926258087158, -0.07070871442556381, 0.1562303751707077, 0.13535986840724945, 0.3867402970790863, -0.3905799090862274, -0.40194225311279297, 2.0978832244873047, -1.0305684804916382, 0.4177711009979248, 0.2730181813240051, -1.4896105527877808, 0.6586450934410095, 1.8983873128890991, 0.27861905097961426, 1.0851885080337524, -0.008533860556781292, -0.3563727140426636, -0.2900755703449249, 0.6031640768051147, -1.1771965026855469, -0.8666424751281738, 0.1130501851439476, 0.7426039576530457, 1.7084808349609375, -1.934600591659546, -0.9408479332923889, -0.5645405650138855, -2.1673336029052734, -0.06250544637441635, -1.2071014642715454, -1.402404546737671, 0.772983729839325, 1.2712465524673462, -0.8635595440864563, -1.0567089319229126, 0.8156932592391968, 1.5315297842025757], [0.5891594886779785, 0.6298246383666992, -0.8520123958587646, -1.270084261894226, 1.6527080535888672, -0.15277498960494995, 2.173281669616699, 0.3114703297615051, 0.11180286109447479, -0.39002105593681335, 0.22668111324310303, 0.17720723152160645, 1.8909517526626587, -0.7173363566398621, -0.4824674129486084, -2.164160966873169, -0.637537956237793, -2.367856740951538, -0.2952437996864319, -0.014101123437285423, -0.9499626755714417, -0.8368223905563354, 0.7042086720466614, 0.5344977378845215, 0.14401641488075256, 0.2964337468147278, 0.9096669554710388, 1.2350906133651733, 1.2394570112228394, -0.13862557709217072, -0.01786944270133972, 0.9560287594795227, 0.2507995665073395, -2.244602918624878, -1.0154860019683838, -1.937677264213562, 0.03473745286464691, -0.4010012745857239, 0.18320882320404053, -1.9459527730941772, 0.8347151875495911, 0.6128685474395752, -0.6055372357368469, -0.2710530161857605, 0.13442480564117432, 0.4819851517677307, 0.5888282060623169, -0.5785173177719116, -0.7390173077583313, -0.7722538113594055, 0.12136021256446838, 1.2532436847686768, -0.8072003722190857, 1.6806222200393677, -0.5769597291946411, -0.28495779633522034, -0.3614748418331146, 0.9237882494926453, 0.7479385137557983, 1.1851401329040527, 0.36703234910964966, -1.518342137336731, -0.9739375710487366, -0.009151875041425228, 0.3985797166824341, -0.7707499265670776, 0.26564136147499084, 2.690263271331787, -1.4004019498825073, 0.693216860294342, 1.3135720491409302, 0.00397476926445961, 0.5763620734214783, 0.4881746172904968, -0.7511872053146362, 2.261753797531128, 0.17979660630226135, 1.5796153545379639, -0.32423314452171326, 0.5106306672096252, -1.8447023630142212, -0.1899217814207077, 0.10340673476457596, -1.2521135807037354, -0.6591009497642517, 0.10149599611759186, 0.40883341431617737, -0.9380991458892822, -1.2890594005584717, -1.656933069229126, 0.7119375467300415, 1.3348368406295776, -0.31051239371299744, -0.8185584545135498, -0.5347715616226196, -0.30293112993240356, 0.8956692218780518, 0.16649459302425385, 0.23785360157489777, 0.009635079652071, 1.0527536869049072, 0.9581314921379089, -0.35909074544906616, 1.379838466644287, -0.37113794684410095, 0.9883220195770264, 2.267883062362671, -1.526382327079773, -0.21106280386447906, 0.6645028591156006, -0.26820313930511475, 0.7585324645042419, 0.07774434238672256, 0.619756817817688, 1.5264922380447388, -0.850527822971344, 0.3416471779346466, -0.6410987377166748, -0.3481947183609009, -0.846543550491333, -0.5289106965065002, -0.7012616395950317, 1.870060682296753, 0.2156570702791214, -0.9453896880149841, 1.5118117332458496, -0.8240541815757751, -0.1995794177055359], [1.1972877979278564, -1.3702934980392456, 0.2313663810491562, -0.6044065952301025, -0.7269236445426941, -0.08979225158691406, 0.6105325818061829, -1.928799033164978, 0.26023122668266296, 1.096440315246582, -0.6718035340309143, 1.0577189922332764, -0.7684809565544128, -0.20663075149059296, 1.45197594165802, -0.10014548897743225, 0.7166495323181152, -0.6997326612472534, 0.42980703711509705, -0.8619339466094971, 0.19160516560077667, -1.3624153137207031, 2.0573971271514893, 0.11964590102434158, 0.38876861333847046, -0.36507686972618103, -0.3199290931224823, 0.9370667338371277, -0.9953380823135376, 1.5359669923782349, 0.1165454238653183, 0.24528954923152924, 0.9154102206230164, 0.6353185772895813, -0.8605816960334778, 0.5586289167404175, 0.6603812575340271, -0.5151985883712769, -0.22899219393730164, 1.959989070892334, -1.4146605730056763, -0.17980651557445526, -0.30912819504737854, -0.18436892330646515, -0.12517498433589935, 0.6913291215896606, -0.6066616773605347, -0.2240753173828125, 0.7153821587562561, 1.535293698310852, -0.9530704617500305, 0.17036758363246918, 1.6549252271652222, 0.1906203329563141, -0.1173567995429039, 0.8924184441566467, -0.6450251340866089, 0.5200327634811401, 0.4960118234157562, -1.185822606086731, 2.0164148807525635, -1.6949559450149536, 0.9300655722618103, -1.4298487901687622, 0.9941676259040833, -0.718411922454834, 0.7826579809188843, -2.0292489528656006, -0.607064425945282, 1.5081623792648315, -0.18460381031036377, 0.709518313407898, -0.2998961806297302, -0.5100716948509216, 0.3293963372707367, -0.8874030113220215, 0.6396133899688721, -1.4219770431518555, 2.7117583751678467, 1.1753755807876587, -0.6790086030960083, 0.3120299279689789, 0.510269045829773, 0.5035718083381653, 1.048753261566162, 1.0233874320983887, -0.26654553413391113, -0.7815672755241394, 0.3780696392059326, -1.1258352994918823, 0.8172944188117981, -2.667079210281372, -0.9537843465805054, -2.2713797092437744, -0.14749684929847717, -0.15042923390865326, 0.4027325212955475, -0.2225315421819687, -2.0605015754699707, -1.0518486499786377, -1.8895946741104126, -0.6534473299980164, 0.2419520616531372, 0.5728129148483276, 1.5843268632888794, 0.29719269275665283, -0.056264039129018784, 0.31705060601234436, -0.6547372341156006, -0.8120843768119812, -0.2413005828857422, 0.4460011124610901, 0.6073023080825806, -0.29418429732322693, -1.8085300922393799, -1.4261246919631958, -0.5898085236549377, 0.11414946615695953, -0.0885653868317604, -1.4846636056900024, -2.1048743724823, 0.2526301145553589, 0.15329712629318237, 0.9847473502159119, -0.185511976480484, -0.8192562460899353, 1.0790386199951172, 1.0042426586151123], [-0.9202667474746704, 0.2514106333255768, 2.078906774520874, 0.059501614421606064, -0.25207850337028503, -0.34651121497154236, -2.29692006111145, -0.9508634209632874, 0.14881974458694458, 0.053736574947834015, -0.07824287563562393, -1.6972529888153076, -1.6541751623153687, -0.09769032895565033, 0.530383825302124, -0.2337670773267746, 0.8057771325111389, -1.440096139907837, -0.0911608561873436, 1.088362216949463, 0.4097970724105835, 0.961233913898468, 1.3541104793548584, -0.04423084482550621, 1.7953201532363892, -0.20628862082958221, 1.3579096794128418, 0.3571721613407135, -0.9500442147254944, -0.61115962266922, -0.5967671871185303, 0.401160329580307, -0.8249992728233337, 2.4685299396514893, -0.3974267542362213, 0.7645740509033203, 0.7910264730453491, 0.05789787694811821, 0.5241579413414001, 1.38936185836792, -0.937352180480957, 1.5623810291290283, -0.7408499717712402, -0.8247038125991821, 0.7367116808891296, 0.11918507516384125, -0.5525352954864502, -0.3570874333381653, -0.814551055431366, -0.4669487476348877, 1.0711294412612915, -2.2143642902374268, 0.6176648736000061, 0.7098965644836426, -1.3763319253921509, -0.007608878891915083, 0.918857991695404, -1.3462440967559814, -0.7286251783370972, -0.45520123839378357, 1.5677387714385986, -2.387390375137329, 1.437880039215088, -1.0280297994613647, -0.08919989317655563, -0.026424873620271683, 1.2046259641647339, -1.2643481492996216, -1.74008309841156, -0.4965416193008423, 0.5065272450447083, 1.0022906064987183, -0.5385256409645081, 0.13949735462665558, -1.8591363430023193, -0.26200196146965027, 0.09862656146287918, 0.201038658618927, -0.24066081643104553, 0.2676718533039093, -1.5404928922653198, 0.042997851967811584, -2.4931814670562744, -0.22691988945007324, 0.9577805399894714, -0.2046637386083603, 2.3476762771606445, 0.7905167937278748, -0.7864108085632324, 1.1106877326965332, 0.23152272403240204, -0.8719151020050049, -1.1381757259368896, -0.633556604385376, 0.9343156218528748, -0.2799481153488159, -0.19567500054836273, 0.4465041756629944, 2.5717737674713135, -0.15832191705703735, -0.2004796862602234, 0.5188334584236145, 0.730911910533905, 0.6047331690788269, 1.2519527673721313, 1.3309789896011353, -0.6839802265167236, -0.7285401225090027, -0.8185417652130127, 1.0487282276153564, 0.20778678357601166, -0.021571902558207512, -0.7795553207397461, -0.3819566071033478, -1.015586018562317, -0.6566290855407715, 0.9479407668113708, -1.276427984237671, 0.43943947553634644, 0.12208323180675507, -0.48367130756378174, -0.5962545275688171, 0.5452954173088074, -0.6710928678512573, 1.4686815738677979, 0.5040173530578613, 2.2226109504699707, -0.30906298756599426], [0.9331135153770447, 0.4557691216468811, -1.2945401668548584, -0.7098456025123596, -0.7543309330940247, -0.16584086418151855, 0.25131189823150635, 1.485507845878601, -1.0014393329620361, -0.10440987348556519, -0.6088626980781555, -1.0488824844360352, 0.5773040056228638, -0.05499538779258728, 0.9656904935836792, -0.30823028087615967, 0.5601192116737366, 0.36455073952674866, -0.7551243305206299, 0.00277190119959414, -0.13382866978645325, 0.5798338055610657, 1.4564721584320068, 1.8575384616851807, 1.5924700498580933, -0.7505588531494141, 0.25323042273521423, -1.248924970626831, 1.180937647819519, -1.268236517906189, -1.2595179080963135, -0.6923913359642029, 0.8124560713768005, -1.0726765394210815, 1.1315418481826782, -1.9660422801971436, -0.02679106406867504, 0.8846428394317627, 1.2448374032974243, -0.6228415966033936, 0.717480480670929, -0.642956018447876, -0.8497925996780396, -1.2625329494476318, -0.16524069011211395, 0.4748460650444031, 0.6398304104804993, 0.6551713943481445, 2.1518704891204834, 0.3319041132926941, -0.26154613494873047, -0.24233496189117432, 0.038968440145254135, -1.192622184753418, -0.8279262781143188, -0.4184942841529846, -0.6978421211242676, 0.37103670835494995, -0.3185023367404938, 0.13435950875282288, -0.18467006087303162, -0.3634546399116516, -1.4733306169509888, -0.1927613765001297, 1.3047196865081787, -0.1500646471977234, -1.3002245426177979, -0.13216589391231537, -0.7798768281936646, 0.699764609336853, -0.41582298278808594, 0.23919963836669922, 1.764296054840088, -0.2222234457731247, -1.013717532157898, -0.6640564799308777, 0.8786857724189758, -0.9203464984893799, -0.007252153474837542, -1.108206033706665, 0.7324995398521423, -0.5905009508132935, -2.251904010772705, 1.3937071561813354, -0.2025981992483139, -0.10655581951141357, 2.8005385398864746, 0.2766970098018646, -0.02964414469897747, 0.260818213224411, 0.07989548146724701, 2.0071263313293457, -1.0584136247634888, 1.1125761270523071, 1.4734861850738525, 0.5177643299102783, 0.9271161556243896, -0.0037129721604287624, 0.3365832567214966, -0.9786449074745178, 1.109678030014038, -0.9417965412139893, 0.22603926062583923, 0.11673625558614731, -0.6325414776802063, 0.23831164836883545, 2.1065220832824707, 0.4829748868942261, 1.0743420124053955, 1.0292335748672485, 1.2187808752059937, -0.31986626982688904, -1.2520935535430908, 0.09589466452598572, -0.34660160541534424, 0.07137539982795715, -0.9835184216499329, -0.07892697304487228, -0.8265263438224792, -0.6229551434516907, 1.5033916234970093, -1.7114379405975342, -0.9061093330383301, 0.6692169308662415, -0.03264015167951584, 1.6845991611480713, -0.915378987789154, -0.044584859162569046], [-0.8562626838684082, -0.49809587001800537, 2.2001993656158447, 0.5872393250465393, -0.38591697812080383, -1.7313607931137085, 2.3300371170043945, -0.047518182545900345, -0.18211382627487183, 1.9072670936584473, -0.2582969665527344, 1.3004498481750488, -0.20207369327545166, 0.03909678757190704, -0.872347354888916, -0.3101191222667694, -0.531108021736145, -0.7424873113632202, 0.5845911502838135, -0.6442521214485168, -1.6195915937423706, -0.08617433905601501, 0.7160488367080688, 0.38295599818229675, -1.1051024198532104, 0.4939444363117218, 0.07362702488899231, -0.2392750382423401, -2.509068012237549, -0.31310150027275085, -1.5896637439727783, 1.0330853462219238, -1.589492917060852, -0.4291762113571167, 2.1358463764190674, -0.23051123321056366, 0.098435178399086, -0.9187090396881104, 0.1930854320526123, 1.690362811088562, -1.0611803531646729, 0.19210098683834076, -0.01396690122783184, 1.3258565664291382, 0.5107148289680481, 0.27762913703918457, 0.8824654817581177, -0.04925599321722984, 0.8741485476493835, -1.0289520025253296, 0.7905983924865723, -0.5364624857902527, 1.8094654083251953, 1.2742363214492798, 1.378260850906372, -0.447078675031662, 0.6695582270622253, 0.2311033457517624, 0.7427508234977722, -0.9579041004180908, 1.324967622756958, 0.3681451380252838, -0.12708492577075958, 0.8366051316261292, 0.7015288472175598, -0.06410154700279236, -0.023316839709877968, -0.3200889825820923, -0.16185668110847473, 0.42154204845428467, 2.1158483028411865, -0.7074756026268005, 0.08871421962976456, -0.6397951245307922, -2.010309934616089, 1.5990437269210815, 0.47010305523872375, -0.694418728351593, 0.4568416476249695, 0.38638895750045776, 0.5056926608085632, -0.6792073845863342, 0.11917265504598618, -0.2749720513820648, 0.18843238055706024, -0.007379482500255108, -0.41263338923454285, 1.0191794633865356, 1.2978335618972778, 0.13496270775794983, -0.5588333606719971, -0.6944572329521179, 0.38583606481552124, -1.8102784156799316, 0.6854420304298401, 0.28977298736572266, -0.14326445758342743, -0.5859934091567993, 0.8372669219970703, -1.1430578231811523, 1.2937506437301636, -0.41765427589416504, 1.2270735502243042, -0.5875309109687805, -1.4039316177368164, -0.1671510487794876, 0.8560463190078735, 1.0232739448547363, -2.544159173965454, 0.6657765507698059, 0.6472774744033813, -0.03053208440542221, -0.75590580701828, -1.0818755626678467, 0.13320794701576233, -1.695360779762268, 0.9710205793380737, -0.2875099182128906, 1.7413575649261475, -1.6655395030975342, -0.8371896743774414, -1.958474040031433, -1.4959222078323364, -1.3288718461990356, 0.1238982081413269, -1.5069174766540527, -1.4168459177017212, -0.8638015389442444], [0.4877158999443054, 0.2568778693675995, 1.225669026374817, 0.7437798976898193, 1.370952844619751, -0.5154901146888733, 0.4023941457271576, -1.4711167812347412, 0.09969604015350342, -1.254726529121399, 1.83016037940979, -0.6499998569488525, 0.6211798191070557, 0.8063178658485413, 0.2763347029685974, -0.489793062210083, -0.32302138209342957, -0.4436740279197693, 1.0159742832183838, -1.5408960580825806, 1.745683193206787, 0.8248412013053894, -1.0310871601104736, -0.023575447499752045, 1.338263750076294, 0.0339638777077198, 0.2265528291463852, 0.8191887736320496, -0.5086402297019958, 0.14295127987861633, 0.15564998984336853, 0.05271049216389656, -1.7473267316818237, -2.168224334716797, -2.2635910511016846, 0.041672028601169586, -1.2622958421707153, -1.7781504392623901, 2.9206132888793945, 2.124422073364258, -0.20511876046657562, -1.240966796875, -0.7266063690185547, 0.2803165912628174, 1.3374890089035034, 0.8425350189208984, 1.4944415092468262, -0.48886948823928833, -0.4748833179473877, -0.8754039406776428, 0.46581658720970154, -0.7773994207382202, -0.6311383843421936, 0.37773075699806213, -0.14966273307800293, 0.04220294952392578, 1.337132215499878, -1.3982560634613037, -0.6314127445220947, 0.4857817590236664, -0.07911593466997147, -2.3373167514801025, -0.34066471457481384, 0.8908881545066833, 0.1153213307261467, 0.6011437773704529, 0.3693622052669525, -0.5979564189910889, -1.7165075540542603, -1.1862742900848389, -1.6817322969436646, 2.8978800773620605, -0.7941691875457764, 0.016619285568594933, 0.7584126591682434, 0.15968485176563263, 0.7160914540290833, 0.22229482233524323, -0.4096168875694275, -1.7590450048446655, -0.6424883008003235, -0.19279958307743073, -0.1384897083044052, -0.25956183671951294, 0.15670406818389893, -0.7897424101829529, -0.26612019538879395, 0.8643895983695984, -0.5435178875923157, -0.7236987948417664, -0.17477671802043915, -1.163511872291565, 0.4562239944934845, 2.775136947631836, 2.2279934883117676, 0.5389980673789978, -0.3069210648536682, -1.5607092380523682, 1.1765477657318115, 0.8700550198554993, 2.4289958477020264, -0.5052709579467773, 1.2897199392318726, 1.2165688276290894, -1.7221654653549194, -0.7865433096885681, -0.8749645948410034, 0.45909589529037476, 1.0089174509048462, 0.10223545879125595, 0.30876854062080383, -0.27340295910835266, 1.5465056896209717, 0.467221200466156, -0.18090303242206573, -0.05344105511903763, 0.1424902379512787, 1.7373344898223877, -1.504518985748291, -0.3760904371738434, 3.0157437324523926, -0.12997068464756012, -1.8548330068588257, -0.9315193891525269, 0.8731681108474731, -0.6678509712219238, 0.3277360796928406, -0.9651294946670532], [-0.5141242742538452, 0.587672233581543, 0.791607677936554, 0.973520815372467, 0.5799509882926941, -0.12202024459838867, 0.11059065163135529, 0.7988715171813965, -2.6028082370758057, 1.125756025314331, -0.9426653981208801, 1.0262538194656372, -0.8130876421928406, 0.4917544424533844, -1.237593650817871, 1.3100801706314087, 0.5803302526473999, 0.4401150941848755, -0.9384406208992004, -0.9323337078094482, 0.2553660571575165, -0.4725523293018341, -0.13200527429580688, 2.2671005725860596, 0.8026701211929321, -0.20714178681373596, -0.494421124458313, -0.9862253069877625, 1.0993887186050415, 0.23878860473632812, 0.6965901255607605, -0.21711571514606476, -0.30661487579345703, 0.8582425713539124, 1.4151837825775146, 0.31169772148132324, 0.6939143538475037, -0.5377402305603027, 0.18118658661842346, 0.4140728712081909, 1.2855806350708008, -0.5073679685592651, -0.02579338103532791, 1.0305217504501343, -0.4227798879146576, 0.7932450771331787, 0.626581609249115, 0.749218761920929, 0.7180212140083313, 0.7154751420021057, 0.08376675099134445, -0.34086301922798157, 0.17394819855690002, -0.5639476180076599, -1.3391731977462769, -1.0060558319091797, -0.2586655616760254, -1.153747797012329, -2.107604503631592, 0.600217342376709, 1.1055904626846313, 1.930787205696106, 0.0787423625588417, 0.46965792775154114, 0.3679972290992737, -0.32801029086112976, -0.3590472340583801, 0.4260004162788391, 0.7212002277374268, -0.45491087436676025, 1.5238966941833496, 1.0167324542999268, -1.3879095315933228, -1.2725160121917725, -1.098205327987671, 1.5857691764831543, 2.2729263305664062, -1.850028157234192, -0.6249418258666992, 0.03477081283926964, -1.4684759378433228, -0.4709785580635071, -0.5329241156578064, -0.05967412516474724, -0.06128176674246788, 0.10663195699453354, -0.6809373497962952, -0.26493728160858154, 1.525077223777771, -1.1274832487106323, -1.0791680812835693, -1.853623628616333, 0.5321677327156067, -0.13672836124897003, -0.7680638432502747, 0.4682598114013672, 0.4192172586917877, -0.058978646993637085, -1.3673218488693237, 2.0037569999694824, -0.5512902736663818, -1.9743765592575073, 1.1674710512161255, -1.297240972518921, 0.19545476138591766, -0.5479212999343872, 0.4618337154388428, -2.2154040336608887, 1.1652798652648926, -1.5489153861999512, 0.8746902346611023, 2.012418270111084, -0.23442837595939636, -1.2685503959655762, 0.4987443685531616, -0.6213134527206421, -0.599195659160614, -1.3102697134017944, 0.551939070224762, -1.5871845483779907, 1.6546921730041504, 0.17121621966362, -1.0283112525939941, -1.3023797273635864, -1.8349945545196533, 0.5637204647064209, -0.9691489934921265, 0.9376826286315918], [0.850177526473999, -0.07100934535264969, 1.5997843742370605, 0.005635669454932213, 0.29709407687187195, -1.6306217908859253, 1.5100828409194946, 0.0049333954229950905, -1.1589527130126953, -0.0256410613656044, 2.498769521713257, -1.2339156866073608, -0.3944782018661499, -0.7297894358634949, 0.9665231108665466, -2.173617362976074, 1.4964599609375, -0.283174604177475, 0.8227182030677795, -0.976315975189209, 0.3845415413379669, -1.17886221408844, -0.4365936517715454, 1.328060507774353, 0.043903984129428864, 0.7349825501441956, -2.0680034160614014, 0.4622512757778168, -0.4390803873538971, -0.041534725576639175, 0.5947322249412537, -1.3707313537597656, -0.8013280034065247, -0.15246905386447906, 2.256425142288208, 0.8877257108688354, 1.1371839046478271, 0.7086823582649231, -1.4994827508926392, -1.3657747507095337, 0.2614455223083496, 0.29978227615356445, 1.1943117380142212, 0.5230537056922913, 0.3187303841114044, -0.6477568745613098, -0.035589270293712616, -1.6089648008346558, -0.11580954492092133, 0.479849636554718, -0.4375152885913849, 1.0039931535720825, -1.3777012825012207, 0.23093028366565704, -0.3654448390007019, -0.07127343863248825, -0.5515432357788086, -1.1341793537139893, 0.06354129314422607, 0.6766761541366577, -0.7725690007209778, 2.549156665802002, -0.6220768690109253, -0.6878331303596497, -0.0943908765912056, 0.3770087659358978, -0.7711183428764343, 0.8112759590148926, 1.7014614343643188, 0.572422444820404, 2.61370587348938, -0.8728586435317993, -0.9198201298713684, -0.5077314972877502, 0.28118714690208435, 0.5628951787948608, -0.6279158592224121, 0.020373739302158356, -0.18262551724910736, 0.5379666090011597, 0.16592800617218018, 0.010193842463195324, -0.061994198709726334, 0.004493269603699446, -0.15604247152805328, -1.3263477087020874, 0.021450016647577286, -0.38090214133262634, 0.2520686686038971, -1.1118026971817017, 0.001036692876368761, -1.2722573280334473, 0.36328181624412537, 0.5033252835273743, 0.043263029307127, 0.9791232347488403, -0.8946866989135742, -0.0502963624894619, 0.3179113566875458, 0.06611917912960052, -0.0940149649977684, -1.0025471448898315, -1.8736686706542969, 0.5065613985061646, -0.1654098480939865, 0.13000768423080444, -0.27727386355400085, 0.6305091977119446, 0.8180700540542603, -1.404604196548462, -1.8643579483032227, -0.41921308636665344, 2.4890220165252686, 0.16967949271202087, -1.7170038223266602, 0.5181737542152405, 1.2484279870986938, 0.16435761749744415, -0.5663562417030334, 0.6127344369888306, -0.21553027629852295, -0.1395014226436615, 0.5470717549324036, -0.990888774394989, 0.11532627791166306, -0.7851617336273193, 0.07350362092256546, 0.458983838558197], [1.6200772523880005, -2.502869129180908, 0.1928701251745224, -1.0622493028640747, -1.121849536895752, -0.9788612723350525, 2.058583974838257, -0.3912903368473053, -1.3955340385437012, -0.2409924864768982, -0.37501612305641174, -0.5025561451911926, 1.3992125988006592, 0.8107691407203674, -1.8409423828125, -1.430638074874878, 1.015104055404663, -0.004943326115608215, -0.5744035840034485, 0.25889313220977783, 0.3693411648273468, 0.6857978105545044, 1.6676791906356812, -0.6333410739898682, -0.0314742736518383, 1.4087014198303223, -1.1611193418502808, -1.1850454807281494, -0.02131049521267414, 1.0970985889434814, 1.041749358177185, -0.46762752532958984, -0.21333393454551697, 0.0031274768989533186, -1.1633161306381226, 0.05810476467013359, 1.17547607421875, -0.887470543384552, 1.42328941822052, 0.6720922589302063, 0.19415037333965302, 0.6295020580291748, 0.5828370451927185, 2.127108335494995, -0.299216091632843, 1.2694567441940308, 0.7022901773452759, -0.44428256154060364, 0.06546562910079956, 0.5044040679931641, -0.8854926824569702, -0.33779752254486084, -0.2681874930858612, -0.9571719765663147, 1.0510913133621216, 0.36582377552986145, 0.8021533489227295, 0.186197429895401, 0.8117941617965698, -0.5375028848648071, 0.4394243359565735, 0.21066509187221527, 0.4164038896560669, 1.6289581060409546, -1.226058840751648, 0.6976234316825867, -0.1828087419271469, 0.19212259352207184, -1.146985411643982, 0.06857132166624069, -0.44091907143592834, 0.13192927837371826, 1.5406310558319092, 0.9307792782783508, -0.4341500997543335, -1.4705860614776611, 0.04587925225496292, -1.9383373260498047, -0.2362077236175537, 1.5461374521255493, 1.4311466217041016, 1.085756778717041, 0.20713862776756287, -1.018768548965454, -0.26539814472198486, -1.1701349020004272, -0.22332924604415894, 0.26488596200942993, 0.30044734477996826, -0.43989884853363037, 0.9078955054283142, 0.2753286361694336, -2.268216133117676, -0.8037790656089783, 0.5025330781936646, 1.031744360923767, 1.166489839553833, 0.4785614013671875, 1.2709416151046753, 0.09941937774419785, -1.521957516670227, -0.8653162717819214, -0.8715712428092957, 0.37464624643325806, -1.4718964099884033, 1.1989915370941162, 1.4477710723876953, 0.8188611268997192, 0.6370406746864319, -1.1894193887710571, 1.85696542263031, 0.030918966978788376, -0.8331592082977295, -0.3086850941181183, 1.097839593887329, -0.07075856626033783, 1.2233831882476807, -0.7215184569358826, 0.5679660439491272, -1.1896097660064697, -0.295545756816864, -0.8163705468177795, -1.6656750440597534, -0.7379351854324341, -0.3831973075866699, -0.8927940130233765, -0.7679151892662048, -0.3270261883735657], [-0.1434638947248459, 0.03478558734059334, 0.023926621302962303, -2.6107027530670166, -0.7131026387214661, -0.9511865377426147, 0.6347734332084656, -0.6422547101974487, -1.0229942798614502, 2.0261077880859375, -2.0359935760498047, -1.9701515436172485, -0.08627495914697647, 0.6244564652442932, 0.9512367248535156, -0.8669996857643127, -0.04830057919025421, -0.1395580768585205, -0.4089605212211609, 0.48658183217048645, -0.08787237107753754, 0.26021531224250793, 1.8974685668945312, 0.08910571038722992, 0.0006454512476921082, -0.21526727080345154, 0.20035246014595032, -0.18025735020637512, -0.6651676297187805, 0.19542039930820465, 1.965798258781433, 0.216425359249115, 0.062446944415569305, 0.005113063845783472, 0.2860840857028961, -1.4670318365097046, 1.689595103263855, -0.8769784569740295, -1.228563904762268, -0.01914862170815468, -0.9902414083480835, 0.015067395754158497, -0.6922195553779602, -0.21278096735477448, 0.3360278010368347, -1.2344800233840942, -1.260949969291687, -1.135233998298645, -0.11463613063097, -0.08960755169391632, -1.1543986797332764, 2.120271682739258, -0.09369337558746338, 0.4105817973613739, 1.1419882774353027, -2.120985746383667, -0.8871040940284729, -0.8709208369255066, 1.509867548942566, -0.4268661439418793, 1.9008829593658447, 0.0006482956232503057, -1.1442468166351318, -0.637718677520752, -0.21452932059764862, 0.7882042527198792, 0.7176070213317871, 0.05417439714074135, -1.6229875087738037, -0.47490331530570984, -1.0739156007766724, -1.5924137830734253, 1.2425259351730347, 2.0578858852386475, -0.5025575160980225, 1.4508386850357056, 0.8071937561035156, -0.26823922991752625, -2.26816725730896, 0.4340798556804657, -1.1796977519989014, 1.289246916770935, 1.0550172328948975, 0.19607266783714294, -2.5219600200653076, 0.379743754863739, -0.7475796341896057, -0.0726935863494873, -0.42572519183158875, 0.1425880789756775, -0.9264352321624756, -1.463135838508606, 2.0315897464752197, 1.2588168382644653, -1.6653281450271606, -0.4850917160511017, -0.8507136702537537, -0.860331654548645, -0.2364986687898636, 0.915702223777771, -0.9067794680595398, -0.4024834930896759, 0.8179227709770203, -1.0150089263916016, 0.08572877198457718, -1.4933712482452393, -0.44178006052970886, -0.7858129143714905, 0.7372031807899475, -2.050598382949829, -0.3785035312175751, 0.7491431832313538, 0.5928134918212891, 0.14363351464271545, -1.0500929355621338, -0.4183628261089325, -0.029216498136520386, -1.025417685508728, 0.41994717717170715, -0.12483203411102295, 0.31835636496543884, 0.2527090907096863, -0.5283761620521545, 0.9495267868041992, -0.6893914937973022, 1.7666245698928833, 0.42842742800712585, -0.16942669451236725], [0.32559284567832947, -3.2069926261901855, 0.7586715221405029, -1.6808457374572754, 1.3749651908874512, 0.14197133481502533, -1.3335187435150146, 0.7022911310195923, -0.10840439796447754, 1.6132750511169434, 0.434334397315979, 0.5254630446434021, 0.6905080080032349, -2.1754229068756104, -1.2043777704238892, -0.9072308540344238, 0.3766094148159027, -0.6796517372131348, 0.7385916709899902, -0.6053966283798218, 2.1243860721588135, 0.7719448804855347, -1.5046454668045044, 0.6434707641601562, -0.1549271047115326, 1.0720593929290771, 1.2678732872009277, 0.384977251291275, 1.0878734588623047, 0.319172203540802, 0.733086884021759, 1.3635891675949097, 0.6735236048698425, -0.5876824855804443, 0.7559310793876648, -0.5409736037254333, 0.4738020598888397, 0.8865019083023071, -0.7442786693572998, -1.0904455184936523, 2.2069334983825684, -0.3605547547340393, 0.6261866688728333, 1.2648088932037354, 1.2130086421966553, -0.571685791015625, 1.8290108442306519, 0.8194690346717834, -0.31402185559272766, 0.9515164494514465, 0.07923369109630585, 1.2101083993911743, 0.8210982084274292, -0.6174358129501343, 0.8258146047592163, -0.9502490162849426, 0.6195757985115051, 0.4480982720851898, 0.47281137108802795, -0.06683260202407837, 1.3921560049057007, -0.5083502531051636, -1.2936981916427612, -0.6507993340492249, -2.472191333770752, -0.12866443395614624, -0.40004798769950867, 1.556058406829834, 0.17795222997665405, -0.11278601735830307, -1.7198045253753662, -2.1919591426849365, -0.022551847621798515, 1.0474389791488647, -0.5225539803504944, 0.7633996605873108, 1.5621446371078491, -1.44564950466156, -0.15945635735988617, -0.7242334485054016, -0.3433801233768463, 0.3440159559249878, 0.46164578199386597, -0.028916826471686363, -1.1367127895355225, -1.7564083337783813, 2.326200246810913, 2.2217657566070557, 1.608199954032898, 0.05480891466140747, 1.865299105644226, -0.38003093004226685, -0.8688507080078125, -1.5538502931594849, -1.9392116069793701, 1.015571117401123, -0.5367450714111328, 0.24276216328144073, -0.050095707178115845, -0.8472326993942261, -0.23110677301883698, 0.3286229968070984, -0.24859902262687683, -0.45872434973716736, 0.28583770990371704, -0.3287721574306488, -0.36393508315086365, 1.473534345626831, -0.46205654740333557, -1.7218810319900513, -0.6551443934440613, 2.638230562210083, -0.1935388296842575, 0.25034192204475403, 0.3707800805568695, 0.9494815468788147, -0.7576781511306763, -1.7422842979431152, -0.300674706697464, 0.11963842809200287, 1.467484712600708, -0.546644389629364, -1.7993301153182983, -2.191836357116699, 0.14054076373577118, -0.8145001530647278, 0.07336178421974182, -1.0518956184387207], [1.3259540796279907, -0.46886304020881653, 0.9963352680206299, -1.4948554039001465, 0.20053182542324066, -0.2862773835659027, 0.19120654463768005, 0.3636232316493988, 0.6611040830612183, 1.1020828485488892, 1.0964852571487427, 0.569385826587677, -1.1256654262542725, 0.15119026601314545, -0.7034108638763428, -1.8953250646591187, -0.30039846897125244, 0.8108792304992676, -0.9952106475830078, -0.6961703896522522, -0.04476963356137276, -1.8017522096633911, -1.7727434635162354, -1.0460119247436523, -1.4736402034759521, 1.2195042371749878, 3.220217227935791, 0.5137662291526794, -0.30453985929489136, 0.08452893048524857, -0.30369606614112854, -0.37886714935302734, 0.03511252626776695, -1.8280543088912964, 0.31869620084762573, 1.238926887512207, -0.5879043936729431, 0.28200361132621765, -0.15784944593906403, 2.4952712059020996, 0.8915639519691467, 1.336662769317627, 0.4459436535835266, -1.8708686828613281, -0.9364416003227234, 0.575901985168457, -0.41791942715644836, -0.7808595895767212, -0.7992053031921387, -2.908905267715454, 1.299010992050171, 0.29352420568466187, 0.062484242022037506, -1.8526191711425781, -0.14757953584194183, -0.20516222715377808, -0.6868796348571777, -1.7277640104293823, -0.07821132987737656, -1.103277564048767, -0.038167789578437805, 2.1172335147857666, -1.5710501670837402, 1.5400029420852661, -1.0911616086959839, -0.030433334410190582, 0.8281134963035583, -1.496177077293396, 0.19514283537864685, 1.0855185985565186, -0.4295819401741028, 0.42165112495422363, -0.588356614112854, -0.9606025815010071, -0.8920746445655823, -0.12917791306972504, -0.19862762093544006, -0.048264745622873306, 0.3269280791282654, 1.1999599933624268, -1.3769176006317139, -0.9190211296081543, -0.4106813371181488, -0.5224263668060303, -1.8945022821426392, -0.25919872522354126, -0.07321371883153915, 0.6248391270637512, -0.4184967875480652, 0.7717796564102173, -0.27964961528778076, -0.894193708896637, 1.7666994333267212, 0.09497860819101334, 0.08867047727108002, -0.5389329791069031, -0.4587211310863495, 0.6454734206199646, -0.02081906981766224, 0.9380218386650085, -1.0218942165374756, 0.2719208002090454, 2.5969290733337402, -0.12377645820379257, 0.673797607421875, 0.018117351457476616, 1.3651100397109985, 1.6069060564041138, -0.3144601881504059, -1.320393443107605, 0.48096904158592224, -0.017439749091863632, -0.5410400629043579, 0.686305582523346, 0.5736666321754456, 0.33898162841796875, -0.36152225732803345, 2.715886354446411, -0.03734251856803894, 0.34701448678970337, -0.0554235614836216, 0.6438786387443542, -1.624765396118164, -0.7081607580184937, 0.7030084133148193, 0.8171719908714294, 1.5449187755584717, -0.3601996600627899], [-0.3422241508960724, -1.0484273433685303, 0.5386779308319092, -0.35804229974746704, 0.9158275723457336, 0.506883978843689, 1.1386185884475708, -0.7965595722198486, -0.030974816530942917, 0.09695062786340714, 1.7001152038574219, -0.9397687911987305, -0.19949810206890106, 0.40707364678382874, -1.2872405052185059, -0.5188665390014648, 0.21514999866485596, 0.45770782232284546, 0.20358668267726898, -0.05198938399553299, 0.8318654894828796, -0.020268399268388748, -0.6469636559486389, -0.10651684552431107, 1.5443648099899292, -0.8000726103782654, 0.9533582925796509, 1.6627109050750732, 1.0962811708450317, 0.8414578437805176, -1.70476233959198, 0.6337879300117493, -0.9028641581535339, -0.47200068831443787, 0.7897980213165283, 0.012413705699145794, -0.37964192032814026, -0.42528194189071655, -0.9536346793174744, -0.04828221723437309, 0.25319647789001465, 0.021582456305623055, 1.4320487976074219, -0.27289631962776184, 0.7638423442840576, -0.45433107018470764, 0.32777178287506104, 0.16689859330654144, -0.1912619173526764, 0.3778730630874634, -0.31827086210250854, -0.973351001739502, 0.41995513439178467, 1.2506998777389526, -0.6020958423614502, -0.9610090851783752, -1.2214164733886719, -0.5349230766296387, -1.4184490442276, 0.13093534111976624, -0.560685396194458, 0.8069223165512085, -0.2930733561515808, 0.47867026925086975, 0.49639892578125, -0.19479279220104218, 0.5582799911499023, 0.9540094137191772, 0.11684253066778183, 0.15480786561965942, -0.32748943567276, -0.557400643825531, -0.3056713342666626, -1.4983625411987305, 2.3043253421783447, -0.1337301880121231, 0.7857657074928284, 0.42506715655326843, 1.4155336618423462, 0.3199334740638733, -0.550422728061676, -0.7042322754859924, -0.9094478487968445, 0.7989615797996521, -1.802517294883728, -1.3324639797210693, -1.51375412940979, 0.6237918138504028, -0.6121072769165039, 0.28554806113243103, -0.2484472543001175, -0.19398511946201324, 0.07437090575695038, -0.4676527678966522, 0.027975112199783325, 1.600877285003662, -1.4005467891693115, -0.5624998807907104, -1.3851845264434814, -0.6732118725776672, 0.8408776521682739, -1.0550286769866943, -0.26640745997428894, -0.8359549045562744, 0.01364278793334961, 0.4518458843231201, 0.34535783529281616, 0.15374483168125153, -1.0099201202392578, 0.10095579922199249, -0.259406179189682, 1.4464991092681885, -0.2457250952720642, 1.4778193235397339, -1.0691478252410889, 0.034165482968091965, 0.38227424025535583, -0.3517643213272095, -0.007668295409530401, -0.3608549237251282, -0.004508520010858774, -0.655418872833252, 0.7754258513450623, -2.091057538986206, 0.7163437008857727, 2.9123003482818604, -0.18717777729034424, -2.0212106704711914], [0.1295255422592163, 0.025727728381752968, 1.4999046325683594, -0.9172921776771545, -0.49647223949432373, 1.5103594064712524, -0.5688257217407227, -0.33917027711868286, -0.5425384044647217, -0.4253283441066742, 0.14590047299861908, -0.5338345170021057, -1.1002647876739502, 1.2752350568771362, 0.019029930233955383, 1.0491466522216797, -0.45910581946372986, 0.2555931508541107, 0.47008654475212097, 1.5344171524047852, -0.42078980803489685, 0.25392428040504456, 1.5903668403625488, 0.37671834230422974, 0.4724943935871124, -0.6326619386672974, -1.2679344415664673, -0.03307213634252548, -1.6617454290390015, 0.6995481252670288, -0.6435511112213135, 2.03433895111084, -0.6471961140632629, -0.17305539548397064, -0.6257126927375793, 1.9472461938858032, -0.4475027024745941, -1.2192553281784058, -1.9895381927490234, 1.978202223777771, -0.4446767568588257, -1.978864312171936, -0.681441605091095, -0.38052651286125183, -0.3165026009082794, 0.10622365772724152, -0.009019673801958561, 0.9822241067886353, -0.8949317932128906, 0.056376419961452484, 0.4116869270801544, -0.1779167354106903, 1.7519488334655762, -1.612200379371643, -0.6853870749473572, 0.5823386311531067, -1.1242923736572266, 1.759131669998169, 0.17072412371635437, 1.1296052932739258, 1.0874156951904297, -0.1968151032924652, -0.3618282973766327, 0.780326247215271, 1.2536829710006714, -0.5744118094444275, -0.7090657353401184, -1.6313532590866089, 0.7043729424476624, 0.718387246131897, 0.39646485447883606, 0.3369874358177185, 0.9549617171287537, -0.1137109249830246, 1.2454699277877808, -1.9958343505859375, 0.24856413900852203, -0.4118204712867737, 0.11431146413087845, -0.06342137604951859, 0.6712125539779663, 0.02380428835749626, -1.004030466079712, -2.1739165782928467, -0.3837897777557373, -0.7464349269866943, 0.6689980626106262, 0.2183867245912552, -0.023151561617851257, 0.7510741353034973, -0.5098577737808228, 0.1923995465040207, 0.22205306589603424, 0.7330537438392639, -1.0352199077606201, 0.5240283012390137, 1.0847249031066895, -0.6449510455131531, 0.5104984045028687, 0.638809323310852, 0.6949374079704285, 0.10016556084156036, -0.7432932257652283, -0.22387148439884186, -0.6355576515197754, 0.33061841130256653, 0.39408811926841736, 0.9669398665428162, 0.19098900258541107, -0.5643817186355591, -1.9617058038711548, 1.0164028406143188, 0.3297539949417114, 1.135435938835144, 0.18097954988479614, 0.2097703516483307, 0.7876915335655212, 1.174055576324463, -0.1712811440229416, -0.7739726305007935, 0.7260609865188599, -0.5373103022575378, -2.939800500869751, -1.0590883493423462, 1.530755877494812, 0.7471773028373718, -0.00762362452223897, -0.5934205055236816], [2.324793815612793, 1.2349984645843506, -0.8088658452033997, 1.5298783779144287, 0.7184552550315857, 0.6915997266769409, 1.9485613107681274, 0.9158158302307129, 1.7013856172561646, -1.653555989265442, -1.123604416847229, 1.2484647035598755, 0.1136171817779541, -0.14909851551055908, -1.4041407108306885, 0.8759653568267822, -0.4386990964412689, -0.41716355085372925, 1.096096396446228, -0.37374037504196167, -0.6118348240852356, 1.6672781705856323, -0.6780115962028503, 0.49066364765167236, 0.6076313257217407, -1.5206037759780884, 1.5701055526733398, 0.5824025273323059, -0.6696657538414001, -0.12070334702730179, -0.7745432257652283, -1.5858683586120605, 0.49046066403388977, -0.5659847259521484, -0.4239670932292938, -1.2349778413772583, -0.46770697832107544, 0.033332470804452896, 0.20260943472385406, -0.3602589964866638, -0.1458231657743454, -0.4463059902191162, -1.5109238624572754, 1.744886040687561, 1.3075025081634521, 0.056381601840257645, 0.008415058255195618, -0.3031042814254761, -0.6423845291137695, -1.1953046321868896, -0.31339383125305176, 0.2634795606136322, 0.7327013611793518, 0.8888078331947327, -0.9399215579032898, -0.5770494937896729, 0.6292486190795898, -0.1659337431192398, -0.3552817702293396, -1.9555633068084717, 0.9660831689834595, 1.8555073738098145, -1.260047197341919, -1.246465802192688, 1.3593493700027466, -0.813503086566925, 0.023320313543081284, -0.19624172151088715, -0.4992614984512329, -0.8178611397743225, 0.20492658019065857, 0.2376101315021515, -0.7559704184532166, -0.9040763974189758, 1.6674859523773193, 0.5010987520217896, 2.3880090713500977, 0.3351198732852936, 0.4739648103713989, -0.10180950909852982, -0.7190423607826233, -0.11669560521841049, 1.5395994186401367, -0.6901903748512268, 1.2321090698242188, 0.857822597026825, -1.448258876800537, 0.8279076814651489, -0.7986900806427002, 0.28588539361953735, 1.1698429584503174, -0.7143489122390747, -1.291625738143921, 1.1195393800735474, -0.9695481061935425, -0.9116458892822266, -1.2416712045669556, 0.633928656578064, -0.8299103379249573, 1.0706268548965454, -0.04148437827825546, 0.0666511207818985, -1.296661138534546, 0.09362450987100601, 0.516510546207428, 0.9745396971702576, -1.2296086549758911, -0.9795386791229248, -0.8981298208236694, 0.14644424617290497, 0.03311556205153465, 1.3581370115280151, -0.6279401779174805, -1.6218899488449097, 1.9054256677627563, -1.4274089336395264, 0.18885955214500427, -1.9180033206939697, -0.33605512976646423, 0.5494349598884583, 1.0928295850753784, -1.4054325819015503, -1.8949496746063232, 0.2556266486644745, 0.9549838900566101, -0.957542896270752, 1.6365606784820557, -0.7698417901992798], [0.33298707008361816, -0.39756718277931213, 1.8008917570114136, 1.2438087463378906, -0.6453825235366821, 0.08847230672836304, 0.137694850564003, 0.17790406942367554, 0.06360190361738205, 2.3459815979003906, 1.1902512311935425, -0.5851159691810608, -0.7954574823379517, -1.3547683954238892, 0.04337245598435402, -0.7759516835212708, -0.5590149164199829, 0.283080518245697, 0.04516022279858589, -0.1497667282819748, -0.0600602850317955, -0.13704612851142883, 0.9618845582008362, 0.3927536904811859, -0.17433564364910126, -1.0131961107254028, -1.2020477056503296, 0.8391923904418945, -1.1875663995742798, -1.60509192943573, -1.468446135520935, -2.146143674850464, 0.0407208688557148, -0.8114156126976013, -0.7244293689727783, -1.4475011825561523, -0.1154497042298317, 0.00673707528039813, -0.1934041678905487, 1.1621153354644775, -0.08948791027069092, 0.3129936158657074, -0.11975828558206558, -0.15534426271915436, 0.012651621364057064, 0.6835159659385681, -0.48513272404670715, 0.6512572765350342, -0.3125758767127991, -0.067349374294281, 0.29866909980773926, 0.31652355194091797, 0.5432313680648804, -1.05205237865448, 0.9024486541748047, -0.13973622024059296, 0.3428837060928345, -0.700950562953949, 1.8381303548812866, -0.22328965365886688, 1.2490185499191284, 0.7048457860946655, 0.8466477990150452, -0.23771575093269348, 0.3647960126399994, 0.973812460899353, 0.651777982711792, 0.010137946344912052, -0.025091974064707756, -0.035555317997932434, -0.34721019864082336, -0.8547174334526062, 0.20052726566791534, -0.027943748980760574, 0.5624635815620422, 1.6863476037979126, 0.809086263179779, 0.304558664560318, 0.11744380742311478, 0.1231146976351738, 1.0916311740875244, 0.07554159313440323, 0.6559461355209351, -0.022115785628557205, 0.41579848527908325, 0.19027210772037506, -0.4890539348125458, -0.08439353108406067, 0.3675091862678528, -0.19775956869125366, 0.8148524761199951, 0.2244642972946167, 0.0028938017785549164, 1.0244455337524414, 1.0072154998779297, -0.22027531266212463, -0.009949653409421444, -0.5127219557762146, 0.49185827374458313, 1.496839165687561, -0.5032798647880554, -1.0339207649230957, -0.2580092251300812, -0.9405094981193542, 0.8343982696533203, -1.173079490661621, 1.317451000213623, -1.2504290342330933, 0.05225474014878273, 0.5584045648574829, -0.9423114061355591, 0.2361498475074768, 0.31877636909484863, 2.152806520462036, 0.06119948625564575, -0.9872199892997742, -0.0613078698515892, 0.8792238831520081, 0.20427046716213226, 0.6382004022598267, -0.11858067661523819, -0.9370746612548828, -0.24139128625392914, 0.9543238878250122, 0.40752726793289185, 0.2935076951980591, 1.519250512123108, -1.0402694940567017], [0.17449764907360077, -0.42058804631233215, -0.35809847712516785, -0.3214004933834076, -0.6596664190292358, -0.10947554558515549, -0.026965804398059845, 0.4715040922164917, -0.9892628788948059, 0.7105910778045654, -0.7249244451522827, -0.4421699345111847, 0.6709445714950562, 0.35376089811325073, -0.6955176591873169, -0.48133745789527893, 0.7763001322746277, 0.6197896003723145, -0.3739144802093506, -1.7178239822387695, -2.8324410915374756, 2.4387624263763428, 0.9905142784118652, 0.8507689237594604, 0.05314749479293823, -0.5949004292488098, -0.3002679646015167, -0.131034255027771, -0.05945008620619774, -1.0381557941436768, -0.4223875105381012, -0.805518627166748, 0.8578925728797913, 0.00850178487598896, 1.033847451210022, -0.6384449601173401, 0.34174636006355286, 1.150197148323059, 0.5049297213554382, -1.5785715579986572, -1.4758418798446655, 0.08039627969264984, 1.271102786064148, 1.1417336463928223, 1.2780340909957886, -2.0019822120666504, 0.010796720162034035, -0.453429251909256, -1.5993144512176514, -0.04550265148282051, -0.8781289458274841, -1.1417434215545654, 0.5927033424377441, -0.75448077917099, 0.9731408953666687, -0.12231668829917908, 1.232767105102539, 1.0469735860824585, -1.5398634672164917, 0.2604043185710907, -0.27629733085632324, 0.6863060593605042, -0.47359487414360046, 1.4002282619476318, 1.0703437328338623, -0.5703134536743164, -0.7783310413360596, 1.5716874599456787, 1.1205217838287354, -0.7026596665382385, -0.022506173700094223, -0.7640120983123779, 0.964418888092041, 0.7457543015480042, 0.9059738516807556, 0.4851309657096863, -0.6594886183738708, -1.0527175664901733, -0.030531447380781174, 0.806376576423645, 1.270656943321228, -0.9455645084381104, -0.19017843902111053, 0.1267169713973999, -1.0260913372039795, -1.552712321281433, 0.6806191205978394, -0.31563180685043335, -1.5400407314300537, -1.8378565311431885, -0.6085680723190308, 1.3949941396713257, -0.828265905380249, -0.15578024089336395, 1.2917966842651367, -0.9233802556991577, -1.2325738668441772, -0.632895827293396, 0.9644417762756348, 0.4568132162094116, -0.26967284083366394, -0.8197069764137268, -1.3367499113082886, -0.3328876495361328, -0.45405513048171997, 1.0620954036712646, -0.648285984992981, -0.7003623247146606, -0.8386072516441345, -0.624186098575592, -1.0163817405700684, 0.1796664595603943, 0.038912687450647354, 1.379460096359253, 0.9909371137619019, 0.5404417514801025, 0.08026409894227982, -0.8073279857635498, -1.246753454208374, -0.7780182957649231, 0.8665063381195068, -1.2280460596084595, -0.3940367102622986, 0.3792778551578522, 0.7125827670097351, 2.194483757019043, -0.25340205430984497, -0.5145778656005859], [0.9418514966964722, 0.31267866492271423, 1.4775745868682861, 0.43806591629981995, -1.4451225996017456, 0.23058733344078064, 2.5727615356445312, -0.44999080896377563, 1.3655840158462524, -0.8322080373764038, -1.202798843383789, -2.0608863830566406, -0.696684718132019, -1.156009316444397, 1.510896921157837, -1.5276308059692383, 0.49374741315841675, -0.0691540539264679, -1.9903388023376465, -0.47176098823547363, 1.254346251487732, 0.5577583909034729, -1.3628469705581665, 1.7240029573440552, -0.6969087719917297, -0.12831810116767883, 0.06521090120077133, -0.17547617852687836, 0.027588317170739174, -0.8026655912399292, -0.9275552034378052, 2.113532304763794, 2.6241259574890137, -1.6356018781661987, 0.504920482635498, -1.071858286857605, 0.5322747230529785, -0.8235905766487122, 0.4087521731853485, -0.8524725437164307, 1.4889616966247559, 2.119243860244751, 0.34797823429107666, -0.20302435755729675, -0.2818525433540344, 0.02539067529141903, 0.6447258591651917, 1.4048168659210205, -0.7644371390342712, 0.8252883553504944, -0.7797304391860962, -1.495802640914917, 0.9178513288497925, -0.6212079524993896, 0.8160513639450073, -0.09147563576698303, 0.7769513130187988, -0.5852640271186829, -0.10285366326570511, -1.272782325744629, 0.6481123566627502, 0.41246020793914795, -0.839303731918335, 0.3134732246398926, -1.2514857053756714, -0.8150730133056641, -0.945644736289978, -0.885356605052948, -0.5314759612083435, -0.15877079963684082, 1.558059573173523, 0.4363861382007599, 1.0487980842590332, 0.034823108464479446, -0.790683925151825, -1.0491312742233276, -1.4374934434890747, 1.433179259300232, -1.1756020784378052, -0.56447434425354, -0.6448294520378113, -0.611990213394165, 0.2871945798397064, 0.6719071269035339, 2.7470133304595947, 0.2565477192401886, 2.067957639694214, -0.00099614926148206, 0.23996871709823608, -1.847170352935791, 0.10763472318649292, -1.8622691631317139, 0.7839078307151794, 2.104952335357666, 0.4053041934967041, -0.5875129699707031, -0.5976001620292664, 3.176804304122925, -0.641869843006134, -0.5030450224876404, -0.5780019164085388, 0.5022585988044739, -1.0528888702392578, 0.23389793932437897, -1.550356388092041, -0.17585472762584686, -1.2671245336532593, 0.42289361357688904, -0.03237922862172127, -0.7981405258178711, -0.8680889010429382, 0.5155016183853149, -0.8972684144973755, 0.05130363255739212, -1.4426809549331665, 0.10217752307653427, -1.0294413566589355, 1.8834056854248047, 0.514266312122345, -0.4663086533546448, 0.5344732403755188, 1.2886828184127808, -0.05772772803902626, 0.3566615581512451, 2.3473851680755615, 0.7108229398727417, -0.6168459057807922, 0.6276038289070129], [-0.3394218981266022, 0.4909703731536865, 1.6079679727554321, -0.24097588658332825, -1.6780507564544678, -2.3213768005371094, 1.0202101469039917, 0.061085090041160583, 1.593430519104004, -1.0606328248977661, -0.45943424105644226, 2.0637600421905518, 2.190786600112915, -0.239850252866745, -1.5169143676757812, 0.6977807283401489, -0.5842373371124268, 0.8975380063056946, 0.7621697783470154, -0.21922282874584198, -1.2274972200393677, 0.5269240736961365, 0.5554114580154419, 0.40269097685813904, -1.447004795074463, -0.9705017805099487, -0.8128664493560791, 0.6770316362380981, 2.6066813468933105, -0.12749813497066498, -0.11890242993831635, 1.056465744972229, 0.3957377076148987, -0.1365700364112854, 0.8904101252555847, -0.5150516629219055, 0.374024361371994, -0.14439357817173004, -0.7559592127799988, -0.7137715816497803, -1.3631503582000732, -0.34707963466644287, 0.8043672442436218, -2.0406241416931152, -0.310844749212265, 0.014868550933897495, 1.0542197227478027, 0.4938514232635498, -0.741632878780365, -0.8042356967926025, -0.73531174659729, -0.18063752353191376, 0.6179353594779968, -0.37099677324295044, 0.39170119166374207, 0.5856470465660095, 1.0762574672698975, 1.3581215143203735, 0.4448027014732361, 0.3535093069076538, -1.2340927124023438, -0.04753313958644867, 0.17965003848075867, 0.790175199508667, 0.6422736048698425, 0.6283098459243774, 0.47892090678215027, -0.2524223029613495, -0.2945346236228943, 0.5170753002166748, -2.1421194076538086, -0.14211571216583252, -1.409616231918335, -0.04447554796934128, 1.2620521783828735, 1.002546787261963, -0.5396903157234192, 0.5960617661476135, 1.318778157234192, -1.6152799129486084, 0.5056395530700684, 1.2034400701522827, 0.9417665600776672, 0.307820588350296, 0.30058205127716064, 0.9029984474182129, 1.3970580101013184, 1.1783630847930908, 0.5464086532592773, -0.07386581599712372, -0.044683005660772324, 0.43874627351760864, -0.13529913127422333, 0.5006300806999207, -0.29931116104125977, 0.35510876774787903, 0.15806061029434204, 1.903953194618225, -0.8891941905021667, 0.8096978664398193, -0.4936264157295227, -0.12399277091026306, 0.6614392399787903, -0.3430190980434418, -2.9301724433898926, -1.247423529624939, -1.375345230102539, -1.0961321592330933, -0.11409423500299454, -0.01753068156540394, -2.4854066371917725, 0.038561660796403885, -0.4927554428577423, 0.4726150929927826, -0.9916999936103821, -1.137372612953186, -0.1473761945962906, -1.661878228187561, 1.2213811874389648, 0.829614520072937, 2.007671594619751, -0.5067381858825684, -0.6347163319587708, 0.708502471446991, -1.8411321640014648, -0.35604792833328247, 1.0958266258239746, -0.8185522556304932], [-1.9623222351074219, -0.8161748051643372, 0.38723617792129517, -2.2224373817443848, -0.9525051116943359, -0.46670934557914734, -0.39721038937568665, -0.5807802677154541, 1.1206448078155518, 0.2029733508825302, 0.02875634841620922, -1.696415901184082, -0.28417348861694336, -0.6581710577011108, 2.125197649002075, 0.698691725730896, -0.3556308448314667, 1.0167427062988281, -1.7459036111831665, 1.1036924123764038, -1.5069308280944824, 0.9449304342269897, -1.8933707475662231, -1.2452528476715088, 1.057861089706421, -1.1986721754074097, -1.7062175273895264, -1.852461338043213, -1.5401983261108398, 0.751668393611908, -0.9803422689437866, -1.008901834487915, 2.314018726348877, -0.03552466258406639, -0.12752491235733032, 0.7014623284339905, 0.8336613774299622, -0.4075104594230652, 0.5055638551712036, 1.4998266696929932, -0.2909395396709442, -0.24787218868732452, -0.7563929557800293, 0.23341260850429535, -2.095113515853882, -0.4684109687805176, -1.94840407371521, 0.6540287733078003, -1.1884593963623047, 0.640013575553894, 1.3973392248153687, -0.8660524487495422, -0.6628223061561584, -0.2970806062221527, 0.23156972229480743, -1.4343513250350952, -0.34380778670310974, -0.8834982514381409, 1.3709856271743774, 0.6208601593971252, -0.39678919315338135, -1.2673925161361694, 1.971327304840088, 1.2419453859329224, -0.3282080292701721, 1.755052089691162, 0.16061195731163025, -2.0835776329040527, 0.8297240138053894, 0.04458416998386383, 0.7864863872528076, 0.13529200851917267, -0.24075713753700256, -1.8803170919418335, 1.0241963863372803, -0.5825755596160889, 1.1658525466918945, -1.6881282329559326, 0.01892426609992981, -0.9242952466011047, 1.1720083951950073, 0.9668518304824829, 0.260159432888031, 2.6610989570617676, 0.6777785420417786, 2.3781261444091797, 0.1249580904841423, 0.3990309238433838, 0.18999403715133667, -1.0424766540527344, -0.7544176578521729, 0.16357293725013733, -0.07452049851417542, -0.7197777032852173, 0.3391772210597992, -0.07839503884315491, 0.6643289923667908, -0.3431321084499359, -1.2412315607070923, -1.513963222503662, -1.273129940032959, -0.3973549008369446, 0.8813380599021912, -0.6406447291374207, 0.710952639579773, 0.25259625911712646, -1.6562327146530151, -0.17893806099891663, -1.0137600898742676, 1.3092643022537231, -0.1291685253381729, -0.3743002712726593, -0.14036132395267487, -0.2038399577140808, 0.799121081829071, 0.07066088169813156, 0.07233522087335587, 0.6227564811706543, 1.1589269638061523, 0.289545476436615, -1.4287693500518799, -0.2849096953868866, 0.5000072717666626, -0.18254928290843964, -1.0247976779937744, -1.353000283241272, 0.7316323518753052, -1.9208157062530518], [-0.24187994003295898, -1.1663753986358643, 0.6576030850410461, -1.2568519115447998, 2.0251548290252686, 0.06681890785694122, -0.2585166394710541, -0.3822377026081085, -0.11483972519636154, 0.8395837545394897, -0.6030113101005554, -1.7091901302337646, -1.9011082649230957, 0.841423749923706, 1.2449469566345215, 1.0709376335144043, -1.1741092205047607, -0.5496765375137329, -2.0075511932373047, 0.8492937088012695, 0.3988974392414093, 0.3654673397541046, 1.642512559890747, 0.22657857835292816, 1.427675724029541, 0.21240568161010742, -0.9041662216186523, -0.4786393642425537, 0.9688528180122375, -0.6207171678543091, -1.611840009689331, -0.29603153467178345, -0.30806005001068115, -0.8962386250495911, 0.2455914169549942, -2.001037359237671, 0.3268316388130188, 0.7499051690101624, 0.17003248631954193, -4.045924663543701, -1.8862203359603882, 0.4234234094619751, -1.5594977140426636, 1.588800072669983, 0.8925992250442505, -2.0458950996398926, -1.769933819770813, -1.4568783044815063, -0.06766156852245331, 0.8309690356254578, 0.008522490039467812, 0.2349996566772461, -1.139669418334961, 0.49178406596183777, -0.40085917711257935, 2.3470871448516846, -0.47030410170555115, -0.43724700808525085, -0.46984580159187317, -1.376220464706421, -0.2915321886539459, -1.4355860948562622, 1.1951611042022705, -1.3218227624893188, -0.019933102652430534, 1.893602728843689, 0.7491469979286194, 0.18424968421459198, 0.41185420751571655, 2.0916450023651123, 2.8901596069335938, -0.922329843044281, -0.3775773346424103, -0.11859997361898422, -0.3513840436935425, 0.2628585398197174, 0.7806233763694763, -0.7430657148361206, 0.9575293660163879, -0.8594672679901123, 0.18387213349342346, -0.5302577614784241, 0.09870412200689316, -0.30591875314712524, 1.0850260257720947, 0.30627691745758057, -1.3542193174362183, 0.23684147000312805, -0.9379134774208069, -0.9121152758598328, -1.6874136924743652, -1.381717324256897, -0.6815829873085022, -0.25506001710891724, -0.24350641667842865, -0.1303885281085968, -0.7838118672370911, 0.13609927892684937, 0.5457555055618286, 0.47447332739830017, -0.7640816569328308, -0.30884602665901184, 0.4142025411128998, -0.6591907143592834, 0.6422398090362549, 1.4991106986999512, 0.8901304602622986, 0.9325999021530151, -1.6861138343811035, -0.6164001822471619, 0.30775707960128784, 1.3839627504348755, 1.5166490077972412, -0.7040191292762756, 1.1024936437606812, -1.112505316734314, -0.18756763637065887, -0.7587805986404419, 0.8265836834907532, -2.790602684020996, -0.28490498661994934, 0.3330969512462616, 0.3837645351886749, 0.7116357684135437, -0.3583633005619049, -0.5127884745597839, -0.031973160803318024, 0.7448720932006836], [0.6532180905342102, 1.6827763319015503, -1.266175627708435, -1.505984902381897, -1.1777998208999634, 0.12010639905929565, -0.0838530957698822, 0.8523781895637512, -0.02967192232608795, 0.3728733956813812, -1.610158085823059, 0.2569698393344879, 0.9330841898918152, -1.124373197555542, -0.332680344581604, 0.27229171991348267, -1.9756072759628296, 0.9710624814033508, 0.501808226108551, 0.7680586576461792, -1.177130103111267, 0.26052936911582947, 2.1164159774780273, -0.5147847533226013, -0.6581066250801086, -0.7398517727851868, 0.5188366174697876, 1.722898244857788, 0.888451337814331, -1.4304132461547852, -0.40482455492019653, -0.878840982913971, -0.8811495900154114, 1.8879703283309937, 1.0516544580459595, -0.641635537147522, -0.50025874376297, 0.06327640265226364, 1.076896071434021, 0.11231015622615814, -0.3442024290561676, 0.5310136675834656, 1.1284509897232056, 0.9859080910682678, -0.00921020656824112, -0.06725793331861496, -0.6182279586791992, 0.2932484447956085, 1.0170902013778687, -0.5701255798339844, -0.30915260314941406, -1.453860878944397, 0.4641566872596741, 0.6859620809555054, 0.3502674698829651, -2.5950427055358887, -1.5481107234954834, -1.1189192533493042, -1.0580233335494995, -1.3218393325805664, 0.37936931848526, -1.9763890504837036, 1.7921135425567627, 0.6527624726295471, 0.47161880135536194, 0.888231635093689, 0.4934755563735962, 0.9623773097991943, 0.6956443190574646, -0.9518706798553467, -1.5963865518569946, -0.6748039126396179, -0.09293225407600403, 0.09800926595926285, -1.5255143642425537, -0.599753737449646, -1.9363093376159668, -0.122705839574337, 0.04240158200263977, -0.2531425356864929, 0.9591771960258484, -1.675723671913147, -0.5070800185203552, 1.7639611959457397, 0.1447097212076187, 0.27633315324783325, -0.4993784725666046, -1.6959919929504395, -2.081573009490967, -0.4827296733856201, 1.890043020248413, -1.4715572595596313, -1.6140495538711548, -0.3503938913345337, -1.1637239456176758, 0.16039814054965973, 1.7323899269104004, -1.3641666173934937, -0.41762349009513855, -0.16709449887275696, 1.9042551517486572, -0.6002120971679688, 0.628052830696106, 0.09845861047506332, -0.12622739374637604, 0.6236454248428345, 0.15487238764762878, -0.7321561574935913, -0.1512829065322876, -0.49943122267723083, -0.14724759757518768, 0.4615468382835388, -0.5741880536079407, -0.11645589023828506, -0.3494662344455719, -0.785100519657135, 1.7502580881118774, -0.2618153691291809, 1.6812716722488403, 0.7967262864112854, 0.4361429214477539, -1.2985771894454956, 1.8582459688186646, -0.481916218996048, -0.28509148955345154, 0.5985035300254822, -0.5622817277908325, 1.1435030698776245]], [[1.3588407039642334, -0.8940200209617615, 0.7032580971717834, -0.4060584604740143, -0.7387025356292725, -0.09200561791658401, -1.1053495407104492, -0.10646368563175201, -0.23977868258953094, -0.5783130526542664, 0.8677136898040771, 1.0694655179977417, -0.37183451652526855, -1.3722583055496216, 1.4590933322906494, -1.4110863208770752, 0.8198585510253906, -0.25479596853256226, -0.16503028571605682, -0.0890587717294693, 1.0898165702819824, -0.8994613289833069, -0.06565464287996292, 0.4788726568222046, 0.5831054449081421, -0.0013329458888620138, 0.08345501869916916, -0.37871918082237244, 0.9645920395851135, 0.5682857036590576, 0.8141791224479675, -1.5711746215820312, -0.09075841307640076, 0.031396567821502686, 0.812704861164093, 0.5814991593360901, 0.32762032747268677, -0.33341601490974426, 1.3398585319519043, -1.1444499492645264, 0.09374206513166428, 2.1914076805114746, 0.07085126638412476, -0.2774580419063568, -0.1580071747303009, 0.5716046690940857, -0.7996092438697815, 0.3751969635486603, 2.085592031478882, -0.9269410967826843, 0.7804933190345764, 0.3705233633518219, 0.27359461784362793, -1.5829339027404785, -0.6909723281860352, 1.6748579740524292, 0.07238321006298065, 1.7436116933822632, -0.20450706779956818, 0.6823082566261292, -0.08802427351474762, 0.8731102347373962, -0.08690885454416275, 1.0684716701507568, 0.2945745885372162, 1.5275373458862305, 0.7720266580581665, 0.03754744678735733, -0.015109177678823471, -0.1706475019454956, -0.2577776610851288, -0.6071265339851379, 0.10454276949167252, -0.10154769569635391, 0.9888906478881836, 0.950516939163208, 0.1773560643196106, -0.7878867387771606, 0.5680108666419983, 0.0961962342262268, -0.6823310256004333, -1.8347982168197632, 3.50961971282959, 0.06661128997802734, 0.4492209553718567, -0.3927927017211914, 0.39797720313072205, -0.15555623173713684, -1.4790019989013672, 2.326899290084839, -0.11702420562505722, -0.3773705065250397, -0.7252646684646606, 0.2775074541568756, 0.4076237082481384, 1.1495678424835205, -0.0044878944754600525, 0.7161721587181091, 0.9838883876800537, 0.3242036998271942, -1.871787667274475, -0.4646340608596802, -0.5957186818122864, -0.8328919410705566, -0.08630518615245819, 0.29536086320877075, 0.6780152320861816, 0.6451410055160522, -0.10469714552164078, -0.8383190631866455, 0.3269316256046295, 0.7539685964584351, -0.851345956325531, -1.0417903661727905, 0.20001453161239624, -1.7135969400405884, 1.8971598148345947, 0.6498397588729858, -0.6018008589744568, 0.5485985279083252, -1.2478536367416382, 0.49489226937294006, 0.13734519481658936, 0.5276466012001038, -0.4342477321624756, 0.2222379744052887, -0.8615604639053345, 1.4082186222076416], [-0.9570861458778381, 0.0511261448264122, -2.3458430767059326, 0.21859949827194214, 1.4205764532089233, -1.3929678201675415, 1.1857328414916992, -0.37366020679473877, 1.9342707395553589, 1.2304530143737793, 0.5542507171630859, 0.7156486511230469, 1.1123015880584717, 1.2777254581451416, 0.4364641308784485, 0.779918909072876, -1.6277889013290405, 0.5279845595359802, -0.7310157418251038, -1.2854828834533691, 1.029030203819275, -1.4870339632034302, -0.3029201626777649, -0.16876521706581116, 0.28874725103378296, 0.8843345642089844, 0.2876701056957245, -0.18590669333934784, 1.9189034700393677, 0.358132004737854, 0.41053420305252075, 2.0577311515808105, 1.3777830600738525, -1.4041948318481445, -0.4099092483520508, 0.35181549191474915, -0.8890153169631958, 0.28188326954841614, -2.1116538047790527, 0.6196509599685669, 1.4412932395935059, -1.563517451286316, 0.3660646975040436, -0.09011312574148178, -1.3007631301879883, 0.42677077651023865, -1.707776665687561, 1.0080411434173584, 0.09003225713968277, -0.3613545298576355, -1.378495216369629, 0.794044554233551, 0.6934241652488708, 0.8590596914291382, -0.15857723355293274, -0.3065517842769623, -0.6707712411880493, 1.4311542510986328, -0.1018529012799263, -1.2419183254241943, 0.5117164254188538, -1.1241787672042847, 0.11239601671695709, -0.5286856889724731, 0.18363365530967712, 0.26329225301742554, 0.29750218987464905, 0.04740985110402107, -0.06532439589500427, 1.309916377067566, -0.3644947111606598, -0.7358611822128296, -0.815807044506073, 2.869295358657837, 0.9204905033111572, -0.0036011661868542433, 0.2675279378890991, -2.491382598876953, -0.4612393081188202, -1.1986806392669678, 1.1704938411712646, -0.949627697467804, 0.16596630215644836, -1.3355087041854858, 2.08909010887146, 0.11973174661397934, -0.35035520792007446, -0.6890090107917786, 0.26294371485710144, -1.0610686540603638, -0.1683613657951355, -1.6137197017669678, -1.4073998928070068, 2.348320722579956, -1.613856554031372, 0.5934296250343323, -0.9392083287239075, -1.8147205114364624, -2.3239564895629883, -0.5353448987007141, 2.2120397090911865, 1.4554080963134766, 0.6157513856887817, 1.0098631381988525, -0.6525734066963196, -2.0390729904174805, 0.5959789752960205, 1.4300285577774048, -0.40545278787612915, 2.1610186100006104, 1.0653215646743774, -1.0289826393127441, -1.2801473140716553, -1.2266545295715332, -0.5877463817596436, 0.6625896096229553, -0.9019464254379272, -0.09182002395391464, 0.08721987158060074, 0.6164128184318542, -0.6548107266426086, 0.9949448704719543, 1.2692863941192627, -0.26090192794799805, -0.5515986680984497, -0.9379642605781555, 0.3421038091182709, 1.2758203744888306], [-0.17261868715286255, 0.20043915510177612, -0.3679063022136688, -0.9346374273300171, -1.7733193635940552, 0.8156822919845581, 1.2165781259536743, -0.16506990790367126, -0.22039198875427246, -0.8899297714233398, 0.560794472694397, 0.8934941291809082, -1.7630202770233154, -1.0286003351211548, 0.751985490322113, 0.2609427869319916, -0.7050740122795105, -1.1509392261505127, 0.3127518892288208, 0.9427196383476257, 0.552926778793335, 2.0932204723358154, 0.5466293692588806, 0.02281549572944641, -0.8862991333007812, -2.3437423706054688, -0.9061444401741028, -1.2757856845855713, -0.750063419342041, 0.8469520807266235, 0.388727068901062, 0.9772225618362427, -0.3452570140361786, 1.4560377597808838, 0.4413166344165802, 1.1170868873596191, -0.3674062192440033, -0.6830200552940369, 0.23710840940475464, -0.44692176580429077, -0.527081310749054, 2.069133758544922, -0.8156982660293579, 0.41770246624946594, -0.4477415084838867, -0.5132603645324707, -0.19739432632923126, 0.6912837028503418, -1.0818567276000977, 0.26850900053977966, 0.5200964212417603, 0.04056257754564285, 0.48325926065444946, 0.4937412440776825, -0.7971371412277222, -1.0293731689453125, 0.9790145754814148, -0.6825783848762512, -1.064732551574707, 0.4159749150276184, -0.34315112233161926, -0.16065435111522675, 0.4311656057834625, 1.5385438203811646, -1.246506690979004, -0.22625064849853516, -0.5901479721069336, 0.24806083738803864, 0.19300825893878937, 0.6358871459960938, 1.4257197380065918, 0.3146328926086426, -0.44865667819976807, -1.467775583267212, 0.48842379450798035, -1.0524455308914185, 1.1746467351913452, -0.5099861025810242, 0.7062817811965942, 0.9111464023590088, -1.7030689716339111, -0.9451558589935303, 0.12397190928459167, 1.3392313718795776, 0.37127336859703064, 0.12013638764619827, 0.046194687485694885, -0.722991943359375, -1.199596881866455, -1.3815953731536865, -0.056756943464279175, 0.2814778983592987, 0.2832616865634918, 0.16754555702209473, 0.32801637053489685, 1.2848602533340454, -0.151789590716362, 0.8431714177131653, 0.6917118430137634, -0.15100526809692383, -0.35625943541526794, 0.06628688424825668, -0.1192634180188179, -0.7392281889915466, -0.6841719746589661, -1.446999192237854, 0.14121192693710327, 1.5897924900054932, 0.8144675493240356, -0.5283125042915344, 1.129733920097351, 0.7383137345314026, -1.5385066270828247, 0.5041980147361755, -2.212944269180298, -0.10017142444849014, 0.09350971132516861, -2.6339051723480225, -0.018012847751379013, -0.2304278016090393, 0.29615893959999084, -1.2577067613601685, -0.8707019090652466, -1.4287629127502441, -1.2140569686889648, -0.19114285707473755, 0.25207239389419556, 0.40942251682281494], [0.8261477947235107, 0.9853004217147827, 0.1062726378440857, -0.1731179654598236, 0.5508933067321777, -1.4066393375396729, 0.35766229033470154, 1.3713066577911377, -0.4542192220687866, -0.5630736947059631, -0.10369396954774857, -0.8330469131469727, -1.9450006484985352, 0.04654623568058014, 0.4052262306213379, -0.2959379255771637, 0.32232674956321716, -0.46372440457344055, 0.781370997428894, -0.8972991108894348, 0.691856861114502, 0.5920681953430176, -0.8734692931175232, -0.07340139150619507, 0.5609229207038879, 0.34384623169898987, -0.15543495118618011, -1.5348386764526367, 0.8719362020492554, 1.888881802558899, 1.388547658920288, -1.2041994333267212, -2.008997678756714, -0.49580731987953186, 1.114058494567871, 0.12283523380756378, 1.028516173362732, 0.8846874237060547, 0.7984520196914673, -1.2160353660583496, 0.24376048147678375, -2.535959005355835, -0.3469434380531311, 1.0414584875106812, -0.9981998205184937, -0.25812986493110657, -0.6239657998085022, 0.8122527599334717, 0.4226095974445343, 0.3098393976688385, -0.23513339459896088, 2.277188539505005, -0.05076586455106735, -0.007603050209581852, -1.2495548725128174, -1.65414297580719, 0.6995977163314819, -0.7040262818336487, 1.1070882081985474, -1.1097995042800903, 1.1781724691390991, 0.9822983741760254, -0.9160623550415039, 0.5734253525733948, 0.36948466300964355, -1.9149317741394043, 2.1118063926696777, 0.6420106887817383, 0.8626075387001038, 0.17464037239551544, -1.1467854976654053, 0.05795904994010925, -0.0068886904045939445, -1.3845210075378418, -0.4126032292842865, -0.21327370405197144, -0.5323675870895386, -0.9402385354042053, -0.3390536606311798, 1.0244662761688232, -0.11541064083576202, 1.6763296127319336, -1.6973260641098022, -0.704491376876831, 1.5117342472076416, -1.761282205581665, 0.8942644000053406, 1.0993809700012207, 1.3569550514221191, -0.03240090608596802, -0.5271955728530884, 0.3237333297729492, -0.20231883227825165, 1.3141472339630127, 0.777426540851593, 0.06518781185150146, 1.0681941509246826, 1.9572453498840332, -0.7019614577293396, -0.3389461040496826, 0.9322453737258911, 0.8578674793243408, 1.0508441925048828, -0.6539286971092224, -1.6130980253219604, 1.8687859773635864, 0.4498198330402374, 0.6468101739883423, -0.48389214277267456, 1.32075035572052, 0.12272153049707413, -1.284658432006836, -0.8891342878341675, 1.8067588806152344, 0.8129478096961975, -1.6183338165283203, 0.14490754902362823, 0.3754226267337799, -0.48366233706474304, 1.586320400238037, -0.8776975274085999, 0.09755392372608185, 0.19916903972625732, 1.1772888898849487, 1.2393711805343628, -0.35538867115974426, -1.269670009613037, 1.335431456565857], [-0.003598729381337762, 1.036058783531189, -0.1750335991382599, -1.3173717260360718, -0.20763254165649414, -1.192057490348816, 1.677846074104309, -0.7771520614624023, 0.20908881723880768, -0.49822574853897095, -0.15219265222549438, 1.1501367092132568, -1.4362775087356567, -0.7394533753395081, 1.3513826131820679, 0.5041249990463257, -0.4895818829536438, 0.7455292344093323, -0.44764819741249084, 0.6878475546836853, -0.054266009479761124, 0.6470101475715637, -1.6917128562927246, 1.1649237871170044, -0.9821507334709167, -0.6547773480415344, -0.22297699749469757, 0.1922278255224228, 0.05961838364601135, 0.31791993975639343, -0.338821142911911, 1.4407175779342651, 0.4393041133880615, -1.52442467212677, -0.29170194268226624, -0.153982475399971, -0.2711471915245056, 1.4992363452911377, -1.6759068965911865, 0.9359501004219055, 1.2649246454238892, -0.9207797646522522, 0.4177493453025818, 0.39619001746177673, 1.1767570972442627, 0.4779977798461914, 0.5968721508979797, 1.1379222869873047, -0.9076275825500488, -0.9788743853569031, 0.5072726011276245, -1.3897452354431152, 0.5695138573646545, -1.6109040975570679, -0.9898710250854492, 0.16227911412715912, -0.8772466778755188, 0.8102885484695435, -1.8337152004241943, -1.1542572975158691, 2.1980111598968506, -0.31103867292404175, -0.12372051924467087, 0.0879574790596962, -2.3815572261810303, 0.2514191269874573, -1.5970450639724731, -2.4825520515441895, -0.38663601875305176, 0.5049338936805725, 2.1902153491973877, 0.9401146769523621, 1.0229769945144653, 1.7136098146438599, 0.20188049972057343, -0.4404893219470978, 0.9728792309761047, 0.24924027919769287, 0.3394363820552826, 0.34078550338745117, 0.9859228134155273, 0.09216863662004471, 0.3243654668331146, -0.5276356935501099, -0.8865974545478821, -1.1812207698822021, -0.3368299603462219, -0.9900283217430115, 0.3455866575241089, 2.232011079788208, -0.08750905841588974, -0.45181915163993835, -0.47232118248939514, -0.046078525483608246, 0.7844053506851196, -0.0701790302991867, 0.7773722410202026, -1.7129032611846924, -0.5167950987815857, -2.6896860599517822, -0.6327612996101379, -0.14266827702522278, -0.991930365562439, -0.23438268899917603, 0.8935674428939819, -2.3702871799468994, -0.6537204384803772, -1.520796775817871, 0.9482905864715576, -0.8344704508781433, -0.17404557764530182, 1.4454599618911743, -0.9295166730880737, -2.379185438156128, -0.3645985424518585, -0.27694109082221985, -0.685673177242279, 2.041943073272705, -0.6835929155349731, -0.3207930326461792, 0.5893713235855103, 0.10323252528905869, -1.018722653388977, -1.4920368194580078, -0.2561434507369995, -0.020801886916160583, 0.15557824075222015, -0.4410657286643982], [-0.32721731066703796, -1.5617449283599854, -0.31632983684539795, 0.5397385954856873, 1.5355335474014282, 1.2277077436447144, -0.9840682744979858, 0.0531298965215683, 0.6622178554534912, -0.41903725266456604, 1.134158730506897, -0.10144665092229843, 1.0207067728042603, -0.5743851661682129, -1.4964289665222168, 0.28177377581596375, -0.7775097489356995, 0.6044058799743652, -0.23328667879104614, -1.239615559577942, 0.3301982283592224, -1.4648326635360718, 1.691723108291626, 1.6539793014526367, 0.39160799980163574, 0.07672198116779327, -1.3875279426574707, -0.6347293257713318, -0.842465877532959, 0.18785962462425232, 0.4469718337059021, 0.9329894185066223, -1.7223279476165771, 0.46848294138908386, -0.6798224449157715, 0.07482943683862686, 0.48745325207710266, 0.7411504983901978, -1.882357120513916, -1.2513377666473389, 0.8775248527526855, 1.9325547218322754, -0.06997323036193848, -0.5597315430641174, -0.0826805979013443, 0.8854740262031555, 0.1386559009552002, -1.103879451751709, 0.12081602215766907, 0.21373578906059265, 0.36729681491851807, 0.04491487890481949, 0.4125925898551941, -0.08706915378570557, -1.3797674179077148, -0.05379614979028702, 0.3443841338157654, 3.183534622192383, -0.512876033782959, -0.8275656700134277, 1.072690725326538, -0.7636301517486572, 2.6725921630859375, -0.7697286605834961, -0.7246981263160706, 1.0757355690002441, 0.8494137525558472, -0.2311442792415619, -0.43956834077835083, -0.757767379283905, -0.8569198846817017, -0.4657377302646637, 0.2578531801700592, -0.558523952960968, 1.869315505027771, -0.4930146634578705, 0.5089637637138367, -1.0846999883651733, 0.4130387306213379, -0.1300082951784134, -0.022958777844905853, 1.0311919450759888, 2.435513496398926, 1.3310610055923462, -1.2651004791259766, -1.3723007440567017, -0.04020223766565323, 0.5206061601638794, -0.6938790082931519, 1.310788631439209, -2.1009552478790283, 0.9585485458374023, 0.7104885578155518, 0.015613153576850891, -1.8686553239822388, -1.1079869270324707, 1.9429508447647095, 0.8408849239349365, -0.09050611406564713, -0.5653496384620667, 1.2040035724639893, -0.1981031745672226, 1.1683558225631714, -0.6823285818099976, 0.45320525765419006, 0.5539640784263611, 0.16227875649929047, -0.4506758451461792, 0.7005773186683655, 0.3792673945426941, 0.31128183007240295, 2.6035468578338623, 0.2549251317977905, -1.7620036602020264, -0.9700939655303955, -1.6624653339385986, 0.30724388360977173, -0.39957094192504883, -0.5796273946762085, -0.16168951988220215, -0.8055117726325989, 0.7656619548797607, 0.6081414818763733, 0.6414302587509155, 1.3655813932418823, -0.47428447008132935, -0.10007737576961517, 0.7435048222541809], [0.9298722743988037, 1.6742931604385376, -0.5592643618583679, -1.6910037994384766, 1.8489806652069092, -1.094262719154358, 1.7995177507400513, -0.618152916431427, 0.08893107622861862, -0.7550157308578491, 0.47077852487564087, -1.2605279684066772, -0.22543509304523468, 0.7881015539169312, -0.2639569044113159, 0.031955912709236145, -1.9331998825073242, 0.7696682810783386, 0.5999181270599365, 0.5327279567718506, 0.7583993673324585, -0.15571628510951996, -1.2453006505966187, 1.0789910554885864, -1.4109368324279785, 0.7247337102890015, 1.8213430643081665, 0.949690043926239, 0.6973840594291687, 0.7790740132331848, -0.1592170149087906, -1.0576834678649902, 0.23306970298290253, -0.5423738956451416, -2.1968672275543213, 0.7051211595535278, 1.8137742280960083, -0.4340756833553314, -0.6590945720672607, 1.0741252899169922, -0.2458391636610031, -1.4544092416763306, -1.3330094814300537, 0.8117086887359619, -1.1613234281539917, 0.07737599313259125, 1.0269259214401245, 0.3027192950248718, 0.2341337651014328, 2.8259470462799072, 0.26311707496643066, 0.3794570565223694, 0.8537595272064209, -0.9955430030822754, -0.3687642812728882, -0.34023821353912354, -1.4512922763824463, 0.24541877210140228, -0.6356022357940674, -0.8573484420776367, -1.3627429008483887, -0.17379534244537354, -1.0459320545196533, 0.5316285490989685, 0.3389435410499573, 2.269517421722412, 0.08237893134355545, 0.36498692631721497, 0.914434552192688, -0.18697218596935272, -1.118149757385254, -0.13449415564537048, -1.094791293144226, 0.7233180403709412, 0.6357658505439758, 0.3927607834339142, 0.6672818064689636, -0.39983925223350525, -1.2693090438842773, 1.0744340419769287, 0.0640215203166008, -1.7039368152618408, -1.5569430589675903, -0.7126011848449707, 0.49168241024017334, 0.965319812297821, -1.213629126548767, -0.7141618728637695, 0.4280342757701874, 0.030177004635334015, -0.8808738589286804, 1.1551121473312378, -0.10669058561325073, 0.2842661440372467, -0.7124024629592896, 0.5532975792884827, -1.6882396936416626, 0.8915156126022339, -0.3746233284473419, 0.6186384558677673, -0.04977613314986229, 0.46104753017425537, 2.0704121589660645, -1.0763295888900757, 0.32269272208213806, -0.11508551239967346, 0.5592646598815918, 0.06934776157140732, 0.5281813740730286, -1.4506176710128784, -0.8533346056938171, 0.6949751377105713, 0.9717360138893127, -1.259939432144165, 0.3351733684539795, -1.2656877040863037, 1.3402657508850098, -1.9694044589996338, 0.9470154643058777, 1.1687427759170532, 0.15736062824726105, -1.52658212184906, -0.9693267345428467, -1.1498701572418213, -0.8809652924537659, -1.1259241104125977, 0.7592389583587646, -0.4487086236476898], [0.9399380087852478, -0.36007288098335266, -0.6977975964546204, 1.8690464496612549, -1.1659008264541626, 0.7030801177024841, 2.192063808441162, 1.500997543334961, 0.519305408000946, 2.0909483432769775, 1.273512840270996, -1.9359890222549438, -1.0500491857528687, -0.902418851852417, -1.6662805080413818, 0.4418565332889557, -0.22646793723106384, 0.3678871691226959, -0.5549879670143127, 0.7662323117256165, -2.1142477989196777, 0.46352526545524597, 0.7955989241600037, -0.014310934580862522, -0.17422185838222504, 0.8451220989227295, 1.425441861152649, 1.1596124172210693, 1.0722471475601196, 0.9312906265258789, 1.0468090772628784, -0.45131492614746094, 1.6386799812316895, -1.3528053760528564, -1.3277772665023804, -0.009817736223340034, 0.12109430134296417, 0.010428530164062977, -1.2656221389770508, -2.2104291915893555, -0.041302673518657684, 0.1447490006685257, 0.007336149457842112, -0.703832745552063, -0.35692745447158813, -0.6757875680923462, 0.6725867390632629, -0.9867855906486511, 1.5095303058624268, 0.0108192702755332, -0.6401270627975464, -0.2233353555202484, -1.4309853315353394, -0.02194385789334774, 0.80947345495224, -0.3244199752807617, 1.4822384119033813, 0.2625811994075775, -0.2884472906589508, 0.30633971095085144, -0.3449561595916748, 0.9066895842552185, 1.4137816429138184, 2.00246524810791, 1.3243647813796997, -0.9857863783836365, 0.21256299316883087, 0.3876025378704071, 0.2857241928577423, -0.30394190549850464, -0.8573317527770996, -1.231953740119934, 0.38969728350639343, 0.23670591413974762, -1.3812505006790161, 0.29867541790008545, 0.3924146592617035, -0.09236571192741394, 1.6152687072753906, 0.28748735785484314, -0.42888376116752625, 2.0206246376037598, -0.25472205877304077, 1.07966148853302, 0.059014398604631424, 0.49308252334594727, -0.13196635246276855, 0.21525928378105164, 0.8796159625053406, -0.7017198801040649, 0.22668592631816864, 0.8711309432983398, 1.2322430610656738, 0.920640230178833, -0.33072948455810547, 1.6819456815719604, 2.73384952545166, 0.24650289118289948, 1.4745988845825195, 1.0641670227050781, -0.8873818516731262, 0.05814467743039131, -0.799378514289856, 0.9614651799201965, 0.34682831168174744, 1.1660414934158325, -1.039594054222107, -0.11591167002916336, -0.06423300504684448, 0.9574042558670044, -0.9612298607826233, -0.015741974115371704, 0.2833106815814972, -0.4944816529750824, 0.6132242679595947, 0.22624696791172028, -0.46729016304016113, -1.2868045568466187, 0.7748270034790039, -1.2761338949203491, 1.406599760055542, -0.9385808110237122, 1.6983736753463745, -0.35370370745658875, 1.278886079788208, -0.5480863451957703, 1.1900990009307861, 0.8168410658836365], [-1.0499582290649414, 0.2427392601966858, -0.6595832705497742, -0.4033328890800476, 2.29170560836792, 0.16513770818710327, -0.06594618409872055, 0.6671019792556763, -1.3572654724121094, -0.1068035364151001, -0.008255441673099995, 0.8376089334487915, 0.6456258893013, 0.3967837989330292, 0.4258638024330139, 0.7932211756706238, 0.0957530066370964, -0.3886510729789734, 0.09607475996017456, -0.033550504595041275, 0.7912411093711853, 0.4956003725528717, -0.024434911087155342, 1.9045953750610352, -0.07082835584878922, -0.5923429131507874, 1.7790955305099487, 0.47306933999061584, -0.9531071782112122, -0.5077885985374451, -0.1845889836549759, -2.4932198524475098, -0.9210841059684753, 0.3312142789363861, 2.364763021469116, 0.021439291536808014, -0.5492479205131531, -1.3073227405548096, 1.2888497114181519, -0.7587440609931946, -1.3803375959396362, 2.7350926399230957, -0.5334253907203674, 1.8601875305175781, -0.825465202331543, 1.2599655389785767, -0.8000994324684143, 0.3772536516189575, 0.41698455810546875, 2.350209951400757, -1.2798023223876953, 1.4944911003112793, 0.4053647518157959, -0.2503795623779297, -0.4357587695121765, 0.4497854709625244, -1.0541857481002808, -1.5725263357162476, -0.2904077172279358, 0.48754170536994934, -1.0904816389083862, 0.580985963344574, 0.8337967991828918, -1.7710152864456177, -0.07896429300308228, 1.1825395822525024, -1.8439342975616455, 0.1343563199043274, 0.2907283902168274, 0.8510025143623352, 0.5245332717895508, -0.41160306334495544, -1.0110503435134888, 1.7754004001617432, -1.8488469123840332, -0.6360413432121277, 0.893592119216919, 1.568735122680664, -0.25805315375328064, 0.7853512763977051, 0.6909875869750977, 0.5695207715034485, -1.4056429862976074, 1.007069706916809, -1.5367863178253174, 0.5982729196548462, 0.7594559192657471, -0.4795812964439392, 0.517802894115448, -0.5199058055877686, 0.22526094317436218, 0.6516349911689758, -0.8299660086631775, 0.8479270935058594, 0.208889901638031, -1.6323390007019043, 1.4504191875457764, -1.1830024719238281, -1.4210021495819092, -0.56654292345047, -1.1631944179534912, 0.384823739528656, -1.386240005493164, -0.31262099742889404, 0.5082283616065979, -1.557624101638794, -1.3919628858566284, -2.388975143432617, 0.08363284915685654, 0.30602389574050903, 1.3136683702468872, -0.5681697726249695, 1.0577107667922974, -0.6442722082138062, 1.0666990280151367, 1.47490656375885, -0.9204370975494385, 0.7011523246765137, -0.4801540672779083, 0.5793871879577637, -1.692407488822937, 0.8110135793685913, -0.19449065625667572, -0.6344151496887207, 1.530342936515808, 2.063724994659424, 1.8216005563735962, 0.25959542393684387], [-0.7141460180282593, -1.0977671146392822, 0.22736063599586487, -0.4501512050628662, -0.2626335322856903, -1.6222659349441528, -0.9087308049201965, -0.8128820061683655, -0.8807474970817566, -1.1794061660766602, 0.377475768327713, 1.1295801401138306, -0.09292585402727127, -0.1554197072982788, 1.077959656715393, 1.290254831314087, 0.8050537705421448, -0.010089659132063389, -1.2982155084609985, 0.24123182892799377, -0.9224644899368286, -0.07097122073173523, -0.7510167360305786, -0.6846615076065063, -0.7211248278617859, 0.7836668491363525, 0.6008221507072449, 0.6794331669807434, 0.21881136298179626, 0.02707221545279026, -0.21515429019927979, -0.023565899580717087, 0.5179435610771179, -0.05786178633570671, 0.30959171056747437, 1.6662218570709229, -2.397793769836426, 1.7969260215759277, -1.2738627195358276, -0.31233441829681396, 0.6552245616912842, -0.35343170166015625, -0.045388948172330856, -1.5364965200424194, -0.558178186416626, -0.07623095065355301, -0.16692791879177094, -1.1725441217422485, -0.282215416431427, -0.1153411865234375, 1.4541468620300293, 0.21556606888771057, 0.5902953743934631, -0.7224016785621643, 1.36764657497406, 2.2333791255950928, -0.22613534331321716, 0.07324077934026718, -0.9224085211753845, -0.49767830967903137, 0.8218078017234802, 0.45136016607284546, -0.12253645062446594, -0.17397135496139526, -0.2701978385448456, 1.0412566661834717, 0.3538835942745209, 1.0047823190689087, -0.176590234041214, 1.2291371822357178, -0.34674137830734253, 1.3652299642562866, 0.12090632319450378, -0.7937500476837158, -1.90941321849823, -0.7937694191932678, 0.22623777389526367, -0.773984968662262, -1.1155214309692383, -1.783313512802124, -0.7662214040756226, -2.077990770339966, -0.4144304096698761, -0.029231417924165726, -0.5014539957046509, 0.02228904515504837, -0.7530937194824219, -0.34324827790260315, 0.5301766395568848, 0.9853783845901489, 0.0033746869303286076, 0.23161640763282776, -0.0015309927985072136, 0.5172727704048157, -1.300060749053955, 2.6787729263305664, 1.122288465499878, 0.4700497090816498, 0.21653680503368378, -1.34603750705719, 1.1561486721038818, -0.8467074036598206, -0.8287832140922546, -0.5927228927612305, -2.175541400909424, 1.3649497032165527, 0.19241319596767426, -0.9644137024879456, -0.4917432963848114, 1.0524412393569946, -0.3641625940799713, -0.5868236422538757, 0.367851585149765, -0.9240897297859192, -0.9276381731033325, -1.0017268657684326, 0.45063281059265137, 0.6002012491226196, 0.5585382580757141, -0.2566365599632263, 0.01248872559517622, -0.5544136166572571, 0.28921952843666077, -0.6712510585784912, 0.2724578082561493, -2.190410614013672, 0.01752469316124916, -0.4536321759223938], [0.5836168527603149, -1.128154993057251, -1.1031049489974976, 0.8611152768135071, 0.6168377995491028, -0.6220623254776001, 1.1291083097457886, -0.057799357920885086, 0.24874292314052582, -1.565658450126648, 0.5176620483398438, -0.7409369945526123, 1.0973451137542725, 0.14070704579353333, 0.34468188881874084, 0.035642869770526886, 0.31200873851776123, -1.0571657419204712, -1.835781216621399, -0.2572956383228302, -0.1735038310289383, 1.0309518575668335, 0.6437858939170837, -0.8129041194915771, 2.536977529525757, -0.6571270823478699, -0.2830779254436493, 0.3456454575061798, -0.6039853096008301, 0.697558581829071, 1.9068235158920288, 0.04986194148659706, -0.8345446586608887, 0.08411543071269989, 0.2833521068096161, -2.1160366535186768, -0.7501056790351868, 1.1699466705322266, -1.2642186880111694, -1.4471911191940308, 0.9911054372787476, -0.7488645911216736, -0.624260425567627, 2.597153425216675, -1.3908474445343018, -1.8645055294036865, -0.2150196135044098, 0.6899915933609009, -0.5511293411254883, 0.3017388880252838, 1.1578071117401123, -0.14955948293209076, 0.0958755612373352, -0.9982715249061584, -0.035888414829969406, 0.6458829045295715, -1.0468171834945679, 0.316310852766037, -1.1925139427185059, 0.12371364235877991, 0.5310551524162292, -1.9143328666687012, 0.3155796527862549, -0.09641700237989426, 1.1105570793151855, 1.1594470739364624, 0.6542374491691589, 0.3084663152694702, -0.8263646960258484, 1.2889363765716553, 0.3444778323173523, -1.0139979124069214, -0.7841010093688965, -0.5214411616325378, 0.33817946910858154, 0.544275164604187, 0.3089476227760315, -0.7177550792694092, -0.8217539191246033, 0.4035602807998657, 0.22626732289791107, 0.029756879433989525, 0.11564314365386963, 0.7989474534988403, 0.4237666428089142, -1.292009711265564, -0.7478269338607788, -1.1581509113311768, -0.15521582961082458, -2.4033539295196533, -0.39492684602737427, -0.7159745097160339, -0.7611533999443054, 0.1269957721233368, 0.39518728852272034, 0.7960343956947327, -0.7001839876174927, 0.43360424041748047, 0.816220760345459, 0.3054371178150177, -0.2610296308994293, -0.9299876689910889, 0.11117035150527954, 0.7514256238937378, -1.0444847345352173, 1.1942135095596313, 0.9549906849861145, 0.2947550415992737, -0.9094716310501099, -0.7962592244148254, 1.6455363035202026, -1.7919328212738037, -2.3807015419006348, 1.3355345726013184, -0.5545850396156311, -0.4365355968475342, 1.53496515750885, -0.7318296432495117, -1.5211803913116455, -0.3492181897163391, -0.14350822567939758, 2.1447324752807617, 0.04356677830219269, -0.9341043829917908, -1.205772042274475, -0.20781981945037842, 0.9602448344230652, -1.1131161451339722], [-0.22358138859272003, -0.1183617040514946, -2.1316237449645996, 1.5227426290512085, -0.9222193360328674, -1.1686522960662842, 0.30627110600471497, 2.3425726890563965, 0.02099001221358776, -1.0312166213989258, 0.4909627139568329, 0.47054511308670044, -1.53542160987854, -0.3027242124080658, 1.0991753339767456, 0.8473370671272278, -0.0010653701610863209, 1.17791748046875, -1.1048482656478882, -0.26964229345321655, 0.4612419307231903, -1.276975154876709, -0.9237239360809326, 1.102264642715454, 0.05946648120880127, 0.839760959148407, -0.5504357814788818, -0.998691976070404, 1.0993026494979858, -0.8636772036552429, -0.1588006168603897, -0.5463908910751343, -2.643275737762451, 0.16006320714950562, 0.6213617920875549, -0.7534942030906677, 0.5123482942581177, 0.5830442309379578, -0.5375173091888428, -0.5731896758079529, -0.513900101184845, 1.36271333694458, 0.6388187408447266, 1.8818187713623047, -0.4939071536064148, 3.130908966064453, 0.236570805311203, 0.55120450258255, -3.6620657444000244, 2.2813403606414795, -0.4536725580692291, 0.5953578948974609, 0.715288519859314, -1.4174672365188599, 0.349933385848999, -0.8951317667961121, 1.6217026710510254, -2.0462141036987305, -1.4450515508651733, 0.6574331521987915, 1.2246266603469849, -0.16922394931316376, 1.0509331226348877, 0.41211721301078796, -2.0919482707977295, -0.1220514178276062, 1.2813775539398193, -0.23459786176681519, 0.6771371364593506, 0.996863067150116, -0.6204383373260498, -0.5303179621696472, -0.021346911787986755, -0.3757440447807312, 0.39159685373306274, 0.9914804697036743, -0.07881268113851547, -0.9225859045982361, 0.7585448026657104, 0.3681207001209259, -0.17331860959529877, -0.44098639488220215, -0.5828582644462585, -1.636232614517212, 1.6247016191482544, -1.9151718616485596, 1.15083646774292, 0.5627375841140747, 1.3292850255966187, -1.245121955871582, 0.032697033137083054, 0.0002069323672913015, 1.1573015451431274, -0.437195748090744, 1.4910895824432373, 0.6714972853660583, 0.3571251332759857, -2.1387505531311035, -0.01107695885002613, 0.2535400092601776, 0.4573625922203064, -1.5820326805114746, -0.07463006675243378, -0.11632173508405685, 0.7858901023864746, 0.020891299471259117, 0.8175438046455383, 1.0898826122283936, 2.4165244102478027, -2.2339699268341064, 0.32113319635391235, -2.2602200508117676, -0.8051335215568542, -0.675696849822998, 0.1627354621887207, -1.9957504272460938, -1.0151950120925903, 0.30735212564468384, 0.13146136701107025, 0.4948091208934784, -0.5508290529251099, -0.5937655568122864, 0.938826858997345, 0.9395706057548523, -0.23806297779083252, -0.1471005231142044, -1.9509152173995972, 0.6316834092140198], [1.2860615253448486, -0.719017744064331, 1.6153994798660278, -0.23276785016059875, 0.5374118089675903, 0.39705875515937805, 1.2271896600723267, -0.7095211148262024, -1.6027706861495972, -0.1677137017250061, -0.07994057983160019, 1.1791518926620483, -0.36427822709083557, -1.0459694862365723, -0.10528714954853058, -0.1004665344953537, 1.3105928897857666, -0.7827973365783691, -0.013851855881512165, 0.7460151314735413, 0.9891980886459351, 0.48069536685943604, -1.0243942737579346, -0.5886004567146301, 0.059567756950855255, 0.9536702632904053, -2.345195770263672, 0.6543192267417908, 0.6784248352050781, -0.19553659856319427, -0.2322586178779602, 0.7017198204994202, -1.4453039169311523, -0.58936607837677, 1.442230463027954, -1.4942972660064697, -0.6159688830375671, -2.644068956375122, 0.8283196091651917, 1.6489269733428955, -0.39856839179992676, 0.32295480370521545, 0.959272563457489, -0.8929889798164368, 1.6637952327728271, 0.12567846477031708, 1.1911059617996216, 1.800851583480835, 0.2740510106086731, -0.006348739843815565, -0.88173508644104, -0.3704635500907898, 1.9175037145614624, -0.579763650894165, 0.778331458568573, -0.31246480345726013, 0.8029779195785522, 0.44325339794158936, -0.9558934569358826, -1.1687549352645874, -0.053409844636917114, -0.041327476501464844, 0.15972527861595154, -0.3939393162727356, 0.48492932319641113, 0.03140457347035408, -0.5614432096481323, -0.9519615173339844, -0.11660496145486832, -0.06241246685385704, -1.928873062133789, -1.405686855316162, -0.07213544100522995, -0.19978433847427368, 2.548220634460449, 0.942839503288269, 1.144676685333252, -0.6359777450561523, 0.5572609901428223, -0.8055296540260315, -0.3551105856895447, -0.2099723517894745, -0.6453945636749268, -1.36715829372406, 0.19235622882843018, -0.8442760109901428, -0.5188398957252502, -1.3532596826553345, 0.09917386621236801, -0.8447303771972656, 0.1013326421380043, -1.0535560846328735, 1.7905259132385254, -0.9197837710380554, 0.5609636306762695, -1.142282485961914, 1.5162725448608398, 1.1480028629302979, 1.2652435302734375, 0.39798909425735474, 0.27496570348739624, 0.6537851095199585, -1.1131882667541504, 0.4667273461818695, 1.5520042181015015, -1.1785866022109985, -0.9738984107971191, 0.5422590970993042, -0.3987707495689392, 1.7879911661148071, 0.19664698839187622, 0.611977756023407, 0.046836331486701965, -0.03994505852460861, -0.3146475851535797, 0.5584036111831665, 0.9602489471435547, -0.7831510305404663, -0.2426522821187973, -0.41140902042388916, 0.202482670545578, 1.326059341430664, 0.5068240165710449, -0.6035358309745789, 1.0971770286560059, -1.5484920740127563, -0.2192848026752472, 1.0940258502960205], [-1.8703817129135132, -2.2430408000946045, -0.7139277458190918, -2.011854648590088, 0.8930475115776062, -1.9723790884017944, 1.038271427154541, 1.1315604448318481, 1.6899125576019287, 1.0528321266174316, -1.5699009895324707, 1.9707552194595337, -0.7140870690345764, -1.073169231414795, 1.07454252243042, -0.39449501037597656, -0.1095217615365982, -0.021450232714414597, 1.541140079498291, -1.4704570770263672, 0.41540107131004333, -1.081207513809204, 1.4233461618423462, 0.8357693552970886, -0.2856597602367401, -0.13811445236206055, 0.21354061365127563, -1.2009718418121338, 0.16469813883304596, 0.8317821025848389, -0.4347791075706482, -0.19804330170154572, -0.25730594992637634, 0.31497931480407715, 0.5775864720344543, 1.3433805704116821, -1.5596582889556885, 0.27377939224243164, -0.22048495709896088, -1.5384291410446167, 0.7456725239753723, -0.7124953269958496, -0.08954404294490814, -0.5307155251502991, -1.3539361953735352, -0.08760961890220642, 0.021103380247950554, -0.18280094861984253, -1.1351640224456787, -0.601584255695343, -0.6805339455604553, 1.3613874912261963, 0.7631595730781555, 0.09887619316577911, -2.010817050933838, 0.5414531826972961, 0.5512117743492126, -0.8863449692726135, -0.14535056054592133, -1.9967482089996338, -0.6795923709869385, -0.3391740620136261, 0.3347892463207245, 0.3382495045661926, -0.5887508988380432, -0.2171054184436798, -1.7379640340805054, -0.04581024497747421, 0.10118017345666885, 0.540316104888916, 1.9817506074905396, -0.798653244972229, 0.022692205384373665, -1.8322856426239014, 1.0465208292007446, -0.9921918511390686, 2.1496496200561523, 0.6990950107574463, 2.000575542449951, -0.5226215124130249, 0.18882164359092712, -0.9545392394065857, 0.08349809795618057, -0.619414210319519, 0.01449509896337986, 0.9829825162887573, 0.7571605443954468, 0.035233039408922195, -0.6232199668884277, 0.3687887489795685, -0.5787745714187622, -0.4040778875350952, -1.8464617729187012, -1.8253157138824463, -0.39039483666419983, 0.8751797676086426, -0.21467217803001404, 0.8535378575325012, -1.7387502193450928, 0.21272985637187958, 0.43157652020454407, -1.413360595703125, 0.5197290778160095, -0.027792012318968773, -0.39060795307159424, -1.7830969095230103, -0.20157980918884277, -0.23648878931999207, -1.2442655563354492, -0.7852020263671875, 0.22197756171226501, 0.16249427199363708, -0.14218121767044067, -0.6941887736320496, -0.8971744179725647, -0.24059079587459564, -0.033432383090257645, -1.5877288579940796, 0.4279859662055969, -0.43677932024002075, -0.49173131585121155, 1.3204525709152222, 0.5256929397583008, -0.4262847900390625, -0.6938395500183105, 0.1165902316570282, 0.9268230199813843, 1.3371399641036987], [-0.6501433253288269, -0.8117814064025879, -0.03697875514626503, 0.007387116085737944, 0.32374125719070435, -0.37620416283607483, -0.7365046739578247, -0.09950157254934311, 0.6916996836662292, 1.114831805229187, -0.6138391494750977, -0.2610647678375244, -0.41511350870132446, 0.010817239060997963, -1.3146872520446777, -0.055384762585163116, -0.9494152665138245, -0.8777464032173157, 0.5895849466323853, 0.3190118074417114, -1.914610743522644, -1.094079852104187, -0.610653817653656, 0.3588596284389496, 0.11643752455711365, -0.47704577445983887, -0.4085082411766052, -0.6265508532524109, -0.7964311242103577, -0.1406746655702591, 1.4012010097503662, 0.11219661682844162, -1.2356318235397339, -0.7013530731201172, 0.5276211500167847, 0.24784143269062042, 0.6116700172424316, 1.079026699066162, 0.06784015893936157, 0.4022340178489685, -1.2735084295272827, -0.3986267149448395, 0.2774183750152588, 0.727089524269104, 0.14161263406276703, 0.9108217358589172, -0.7206301093101501, 0.49516478180885315, -0.5907559990882874, 0.28296059370040894, 0.8333449959754944, 2.936257839202881, 0.019651291891932487, 0.1399991810321808, -0.6720290780067444, -0.6573904752731323, -0.13881929218769073, -1.2405200004577637, 0.6527888774871826, -0.3047327697277069, -0.7496454119682312, 0.7783253788948059, 0.4454207122325897, -0.014329837635159492, -2.1461923122406006, 0.9314892888069153, -0.663506805896759, -1.089421272277832, 0.3994719088077545, 0.14162692427635193, 0.7148793935775757, -0.5377371311187744, -0.3323386311531067, -0.7587076425552368, -1.4511420726776123, 0.45990684628486633, 0.5626617074012756, -0.3231087028980255, 1.425423264503479, -0.4049796760082245, 0.8540704846382141, 1.1512985229492188, -0.9639483094215393, 0.3734687268733978, -0.7587775588035583, 0.8208545446395874, 1.443739891052246, -0.09961782395839691, 2.1617321968078613, -0.837207555770874, 0.6430733799934387, -0.5605844259262085, -0.06244155019521713, 1.7982020378112793, -0.30727192759513855, -0.030375288799405098, -1.072085976600647, 0.660743236541748, -0.3980086147785187, -0.5876112580299377, -1.1866700649261475, 0.4625016450881958, -0.2830727994441986, -0.8022581338882446, 0.633924126625061, -0.8794991374015808, 0.9951488375663757, -0.14327356219291687, 0.0924387201666832, 0.16638430953025818, -0.14492832124233246, -0.5179648399353027, 0.7744412422180176, 0.5837156772613525, -0.7631903290748596, 0.09325315058231354, 0.3939112424850464, -0.5673971176147461, -1.1480563879013062, 1.0869797468185425, -0.5421473979949951, 0.6599436402320862, -0.24995295703411102, 0.10203423351049423, -1.4979045391082764, 1.339152455329895, -0.16083325445652008, 0.9881048202514648], [-0.3491973876953125, -2.3479628562927246, -0.6325750350952148, 1.6028099060058594, -1.042635202407837, 0.35021522641181946, 0.22557228803634644, 1.5851689577102661, 1.1324349641799927, -2.2321465015411377, 0.6515328884124756, -0.6059184670448303, 2.4533326625823975, 0.3386111855506897, 0.3859921991825104, -0.6313756704330444, 0.693662166595459, 0.268740713596344, -0.10376259684562683, -0.9167908430099487, 0.10112012177705765, 0.6317650079727173, -0.4082154333591461, -0.4803345799446106, 0.5308681726455688, 2.202244758605957, -2.0904593467712402, -0.6498481631278992, -1.0707694292068481, 1.2254849672317505, -0.050666820257902145, -1.1894932985305786, -0.1571899801492691, -1.2446812391281128, 1.769513726234436, -0.896560549736023, -0.16196049749851227, 0.6856470704078674, -0.30954509973526, 0.12424274533987045, 0.4634671211242676, -0.757161021232605, -0.47863325476646423, -0.8481130599975586, -0.0725771114230156, 1.271754264831543, 0.004832317586988211, -1.8923603296279907, -0.8719788789749146, -0.006265800446271896, 2.023758888244629, -0.07877057790756226, -0.2652294635772705, 0.07584606856107712, 1.1780048608779907, 0.7165864706039429, 2.5842299461364746, -0.07542084902524948, 0.9018683433532715, -1.7112245559692383, 1.0248051881790161, 0.10058589279651642, -1.3835253715515137, -0.14323759078979492, -0.7714532017707825, -2.4172418117523193, 0.34599101543426514, 0.7155278921127319, -3.271373748779297, 1.894421935081482, -0.3003315031528473, -0.2024533897638321, 0.37806424498558044, 1.139433741569519, 1.339245080947876, 1.6421840190887451, -1.0087144374847412, 0.917813241481781, -1.9667943716049194, 2.071091890335083, 0.018618488684296608, -0.4526630938053131, -0.9943477511405945, 1.0838662385940552, -0.5865451693534851, 0.6045005917549133, 1.248846173286438, -0.9227375388145447, 0.1738310009241104, 1.004402756690979, 0.7686073780059814, -1.09768545627594, 1.2600051164627075, -0.5550209879875183, -0.23889359831809998, -1.5021406412124634, -0.5166749954223633, 2.043621063232422, -1.4024049043655396, 0.12749892473220825, -1.260758399963379, 0.044965021312236786, 0.03047621250152588, -0.5652186274528503, 1.3039895296096802, 0.6688781976699829, 1.159746766090393, 0.4955715835094452, -0.2582080662250519, -1.4519116878509521, 1.2086058855056763, 0.0448094867169857, 1.3716411590576172, 0.9427540302276611, 1.3171309232711792, 0.46480074524879456, 0.6374551653862, -0.9921563267707825, 1.3499253988265991, 0.48962804675102234, -0.38372373580932617, -1.3287936449050903, 0.367576003074646, 0.5141083598136902, 0.36577120423316956, -1.4147565364837646, -0.6946606636047363, -0.316569060087204], [0.3226048946380615, 0.6934868097305298, -0.5917320251464844, 0.2383369356393814, -0.5881180763244629, 0.6564033031463623, -0.6433175802230835, -0.4140585958957672, 0.6420994997024536, -0.1891796588897705, -0.9663764834403992, 0.9386327862739563, -0.39183536171913147, -0.6783753633499146, -0.11243488639593124, -0.09499749541282654, -0.7807394862174988, 0.30619582533836365, -1.2066987752914429, 0.298174113035202, 0.17173480987548828, -2.16464900970459, 2.7425143718719482, -0.6454351544380188, -1.198150396347046, -0.24307310581207275, -0.7171815633773804, 1.4087858200073242, 1.7488330602645874, -1.924724817276001, 0.8713603615760803, -0.31523072719573975, 0.4167386293411255, 1.1344308853149414, -0.8745328783988953, -0.8669595122337341, 0.9555251598358154, -0.5568081140518188, 1.1908156871795654, -0.08378029614686966, 0.3509504199028015, -1.5614473819732666, -0.471544474363327, -0.5001476407051086, -0.22046631574630737, 1.025090217590332, 0.5559088587760925, -0.40646660327911377, 0.40361398458480835, 0.17925667762756348, -1.0856149196624756, -0.24375516176223755, 0.5017092227935791, 0.5682069659233093, 1.9645702838897705, -0.650912880897522, -1.2206730842590332, -0.7098140716552734, -2.1120083332061768, -0.08269061148166656, 0.39152929186820984, -1.0352051258087158, 1.416383981704712, -1.3489629030227661, -0.5086917877197266, 0.019702184945344925, -0.15920554101467133, 0.2627093195915222, -0.9527045488357544, -0.19090589880943298, -1.0858957767486572, 0.3739020526409149, 0.9925050139427185, 0.849368155002594, 1.155111312866211, 1.6596037149429321, 1.0788640975952148, 0.3516790270805359, -0.2175932377576828, -1.637886881828308, -0.32121342420578003, -0.3075866401195526, 0.1489398181438446, 0.33674976229667664, 0.5082732439041138, -1.9528096914291382, -0.22471080720424652, -0.14547286927700043, -1.513149380683899, 0.1364818513393402, 0.1076737716794014, 0.2429778277873993, -0.4951459765434265, -1.3564213514328003, -0.5600216388702393, -0.14075620472431183, -0.5407671928405762, -0.18206453323364258, -0.6696131825447083, 0.5109902024269104, -0.7654649615287781, 0.11414282768964767, -0.8301755785942078, 0.03485669195652008, -0.5980918407440186, -0.2652932405471802, -0.19047582149505615, 0.33937641978263855, -2.033862590789795, -0.1012415960431099, -0.7548924088478088, 1.4770088195800781, 0.2390999048948288, -0.8963412642478943, -1.1798601150512695, -0.9962424635887146, -0.6442714333534241, 0.34027910232543945, 0.47903138399124146, -1.1391717195510864, 2.1698126792907715, 0.24915015697479248, 0.7372345328330994, -0.33462825417518616, 0.4616248905658722, 1.3926920890808105, -0.06789993494749069, 1.7279117107391357], [-1.0609687566757202, -1.2229585647583008, -0.5841703414916992, 1.955410122871399, -0.562728226184845, -1.0041639804840088, 0.2097485214471817, -0.6364578008651733, -1.3118255138397217, -0.2742322087287903, 1.7017625570297241, -0.9824084639549255, -0.05534027889370918, 0.4631693363189697, -0.25905054807662964, -0.1711927354335785, 0.19639720022678375, 0.3717728555202484, -0.36024099588394165, 0.6279224157333374, 1.0311065912246704, 0.2686379849910736, 0.26118791103363037, 1.0300699472427368, 0.6172347068786621, 0.32809656858444214, 0.06517432630062103, -2.026693105697632, 1.110632061958313, -0.7725380063056946, -0.6922124624252319, 0.028983479365706444, -0.683198869228363, 0.617273211479187, -0.6346297860145569, -0.31244245171546936, 1.1567996740341187, 1.4837324619293213, 0.22107289731502533, -2.7911529541015625, 0.7721736431121826, -0.2039104402065277, -0.6346948146820068, -0.3325270712375641, 0.48373037576675415, -1.2242639064788818, 0.7442262172698975, 0.8904263377189636, 0.9181365370750427, -1.3242707252502441, -0.30202457308769226, 0.6025513410568237, 0.665610134601593, -1.737484335899353, -1.1017838716506958, 0.9133514165878296, 0.6778876781463623, -1.4545973539352417, 0.956509530544281, -0.2957856059074402, -1.8200753927230835, 0.49016743898391724, 1.0898317098617554, 0.7039074897766113, 0.7292991876602173, 0.750986635684967, 0.9006456732749939, 0.5546162724494934, -0.3444381654262543, 2.021376848220825, -0.11486195027828217, -0.6493066549301147, 2.5246338844299316, 1.620969533920288, -0.856320858001709, 0.6140272617340088, -0.24538709223270416, -0.5699029564857483, 0.18882225453853607, -0.436681866645813, -0.7289319634437561, -1.2900614738464355, -0.568690299987793, 0.05537943169474602, -1.550490379333496, 1.9290735721588135, -1.2116084098815918, -0.3594675064086914, 0.41315996646881104, -0.19252555072307587, 0.39871057868003845, 0.40711548924446106, -0.5608925819396973, 0.9989638924598694, -0.4394744038581848, 0.3569488525390625, 0.06739514321088791, -0.14874789118766785, -1.3024729490280151, -0.38264816999435425, 0.11865963786840439, -0.9419251084327698, 0.4820505976676941, 1.2077782154083252, -0.16237568855285645, 0.3660223186016083, 1.5504363775253296, 0.8743063807487488, -0.48469677567481995, -0.3257778584957123, 0.5467629432678223, -0.047893624752759933, 0.39909714460372925, -0.3091304898262024, -1.0929598808288574, -0.31033170223236084, -0.891972541809082, -1.1352676153182983, -2.1605629920959473, -0.2159658670425415, 0.780306875705719, 0.7153477072715759, 1.9128196239471436, 0.5024290084838867, 0.7068727016448975, -0.4852840006351471, 0.8581960201263428, 0.9864311814308167], [-0.21962977945804596, 0.24404524266719818, -0.8830124735832214, 0.8687208890914917, -0.17849983274936676, -0.3978710472583771, -0.30659210681915283, 0.3868371844291687, -1.2116873264312744, -0.2774781882762909, 1.0780714750289917, -2.1910595893859863, -0.22772204875946045, 0.040494404733181, 0.42338424921035767, -3.860642433166504, -0.8259068131446838, 0.24477629363536835, -1.187741756439209, 1.0696179866790771, 0.5147005319595337, -1.1041187047958374, 0.8880239725112915, 0.060606762766838074, 1.594239592552185, 0.13187061250209808, 1.9664469957351685, -0.02428027242422104, 0.3892485201358795, 1.0113025903701782, 0.9571290612220764, -1.0239543914794922, -0.10198422521352768, -1.644736886024475, 0.7268122434616089, -0.10702542215585709, -0.20782914757728577, 1.2235349416732788, -0.5397193431854248, 0.29976993799209595, 0.12113595753908157, -1.0323710441589355, -0.09619531780481339, 0.6903000473976135, 0.1754245012998581, 0.11946068704128265, 2.376464605331421, 1.0308585166931152, -0.6370733380317688, -0.8319075703620911, 0.38780948519706726, 0.09852763265371323, -0.7956579923629761, -0.9182732701301575, -0.44075992703437805, 0.8695693016052246, 2.2628068923950195, -0.33359676599502563, 0.16507476568222046, -0.3715975284576416, 0.45214730501174927, 1.637435793876648, 0.3155534267425537, 1.1976797580718994, -1.5208312273025513, 0.29482361674308777, 0.40563955903053284, -0.9804788827896118, 0.3630652129650116, -0.16857631504535675, -0.092491015791893, 1.3678361177444458, 1.2067075967788696, -0.09423292428255081, 1.8114262819290161, 0.2233681082725525, 0.8205187320709229, -1.6953575611114502, 0.524630606174469, 0.9513651728630066, -0.1573610007762909, -1.6077439785003662, -0.33331042528152466, -0.45336341857910156, 1.9005458354949951, -0.406660258769989, 0.2464364618062973, 0.8390951752662659, -0.2828656733036041, 0.11070065200328827, -0.31588196754455566, 0.32650595903396606, -1.413849949836731, 0.427161306142807, -1.1665080785751343, -0.5639309883117676, -0.4368249475955963, 0.4869610071182251, -0.7901941537857056, 1.2748289108276367, -0.8676218390464783, 0.7414213418960571, -2.117823362350464, 1.5140807628631592, -0.13607686758041382, 1.0977349281311035, -0.09561489522457123, 1.264244556427002, 1.028152346611023, 0.19574551284313202, -0.4973752200603485, 1.801662802696228, -0.840362548828125, -1.0309935808181763, -0.9273748397827148, 1.2877143621444702, 0.23966878652572632, -0.485564649105072, -0.646969735622406, -0.5150206089019775, 2.490398406982422, -0.45055878162384033, 0.4416687786579132, 0.1342444121837616, 0.334548681974411, -0.32168328762054443, 0.6131629347801208, -0.2473393678665161], [0.16583633422851562, 0.4476228356361389, 0.5075432062149048, -0.23515501618385315, 0.01804172806441784, -0.3044077455997467, -0.4557139575481415, 0.11632738262414932, -1.3204041719436646, -1.27154541015625, -0.5665408372879028, 1.2854323387145996, -0.39314863085746765, -1.690415620803833, 0.5200285911560059, -1.366096019744873, -1.5243096351623535, 0.012692691758275032, -0.2963091731071472, 0.7039098739624023, -1.375807285308838, -1.7043570280075073, 1.7417242527008057, 1.1873724460601807, 1.181105375289917, 1.1520617008209229, -0.26742538809776306, -1.8122977018356323, 0.24266639351844788, 1.4819064140319824, 0.409538596868515, 1.1294571161270142, 2.3355302810668945, -0.23626267910003662, -0.9677813053131104, 1.5033321380615234, 1.2899737358093262, -0.73458331823349, 2.002635955810547, -0.17831192910671234, -0.4732655882835388, 0.8676438927650452, -0.27969998121261597, -0.9633967280387878, 0.6476088762283325, -0.2029375433921814, 0.3304762840270996, -0.7859506607055664, 1.0866646766662598, 0.7535648345947266, -0.6665419340133667, -0.37398603558540344, 0.09992137551307678, -0.10151206701993942, 1.1576002836227417, 1.7149670124053955, 0.5644971132278442, 0.5512511134147644, 3.1734132766723633, -1.2162271738052368, -0.729638397693634, -2.002729654312134, -0.7026578187942505, 0.4457698464393616, -1.6396727561950684, -0.24002696573734283, -0.6959059238433838, 0.47430577874183655, -1.71418035030365, 0.274544358253479, -1.5649347305297852, 0.5678244233131409, -1.4481821060180664, 1.1046233177185059, 0.18273548781871796, -0.3547581434249878, 0.10352209955453873, -2.508904457092285, -2.338933229446411, 0.09273336827754974, 0.41247886419296265, -0.6220442056655884, -0.6399856805801392, -0.43552452325820923, -0.43342795968055725, -0.5883519053459167, 1.004604458808899, -0.05647886171936989, -0.07529299706220627, 0.3422999083995819, 1.3826310634613037, 1.5237501859664917, -1.2538011074066162, -1.5589613914489746, 0.6706879734992981, 0.2592112720012665, 0.48269030451774597, 0.2885397970676422, -1.1187742948532104, -0.2702002227306366, -1.211061716079712, -2.238562822341919, -0.28500136733055115, 1.5636128187179565, 1.07741117477417, 1.4513099193572998, 0.566331684589386, -0.2198483645915985, 0.045831695199012756, 0.41029444336891174, -1.258497714996338, -0.4296018183231354, 0.4080638885498047, 2.096266269683838, -1.1193195581436157, -0.12756548821926117, 0.3173481225967407, -1.762343406677246, -0.8567941784858704, -0.22172191739082336, -0.21620050072669983, 0.535780131816864, -2.401257038116455, -1.0946494340896606, -0.8711017966270447, -1.4854357242584229, -0.6755560040473938, 1.7261074781417847], [-1.7954481840133667, 1.484349250793457, -1.731064796447754, 1.944510817527771, -0.8788377642631531, 2.783632278442383, 0.8317379355430603, -1.6114634275436401, 0.8840659856796265, 0.09935358911752701, -1.544273853302002, 0.9849055409431458, -0.29492712020874023, -1.4134743213653564, 0.3879120945930481, -0.23284146189689636, -0.7079224586486816, 0.3473292589187622, -1.247928500175476, 0.43497875332832336, 1.0672845840454102, 0.10821634531021118, 2.226665735244751, 0.17678852379322052, 1.0077334642410278, 1.0629805326461792, -1.4453092813491821, -0.9779694080352783, -1.9838322401046753, -0.5054055452346802, -0.37722787261009216, 0.10907494276762009, -2.426767349243164, -0.28178250789642334, -0.8240077495574951, -0.5962603092193604, 0.8749812841415405, 0.843757688999176, -0.13762140274047852, 0.1489446759223938, 1.2180569171905518, -0.36112645268440247, 0.6789577007293701, 0.035035617649555206, 1.334551453590393, -0.4341966509819031, 0.3589257299900055, 0.7777249217033386, -0.29972103238105774, 1.029524326324463, 0.5262618064880371, 0.24893847107887268, 0.6851567625999451, 0.3247435986995697, 0.8355205655097961, -1.5044865608215332, -1.3307371139526367, 0.3704943060874939, -0.5190008282661438, 0.5805631875991821, 0.4342264235019684, -0.2853703200817108, 0.18451383709907532, -1.4437816143035889, 0.4267735183238983, 0.9505433440208435, -0.7418972253799438, 0.9396001100540161, -0.6861616969108582, -1.6609537601470947, 0.022011753171682358, 1.2651665210723877, 0.174312025308609, -1.1641511917114258, 0.4113091826438904, 0.4330686032772064, -1.2991317510604858, -1.761125087738037, 0.2979820668697357, -0.9777047038078308, -0.7807621955871582, 0.24090039730072021, -0.8015056848526001, 0.038906216621398926, -0.4077395498752594, -1.3431397676467896, 0.14800220727920532, 0.3063068389892578, -1.9421182870864868, -1.100992202758789, -0.12168679386377335, 0.3859602212905884, -0.5777100324630737, -1.0444597005844116, 0.2640382647514343, -0.8292285799980164, -0.32613736391067505, -0.014982704073190689, 0.49346083402633667, -0.6712695360183716, 0.21946987509727478, -0.19988319277763367, -0.9036881923675537, 0.5271732807159424, 0.025603584945201874, -0.39737579226493835, -1.4462600946426392, -0.034698486328125, 0.276328980922699, 0.8162899613380432, 0.17737315595149994, 1.5895769596099854, -0.4856996536254883, 0.9883702397346497, -0.659398078918457, 1.3621885776519775, 0.599221408367157, -0.475927472114563, -2.514979839324951, -0.0683094784617424, -0.8127104640007019, -0.4327841103076935, -0.5805396437644958, 1.0691545009613037, -0.930600106716156, 1.4355322122573853, 0.5997164249420166, -1.6947622299194336], [-1.6448317766189575, 0.8711004853248596, 1.8152532577514648, 1.2426226139068604, -1.241809606552124, 1.1296740770339966, -0.5218451619148254, 0.7656865119934082, -0.8984413146972656, -1.1068320274353027, 0.2720145285129547, 1.6362982988357544, 0.9643903970718384, 0.8196263313293457, 2.1570756435394287, 0.5260711908340454, 0.4794714152812958, 0.4574519991874695, 0.17878803610801697, 1.4471235275268555, -0.6962204575538635, 0.5470154285430908, 0.681045413017273, -1.2263966798782349, 0.01100081205368042, -0.531950831413269, 0.5642398595809937, -0.7723190784454346, 1.2184466123580933, 1.7843027114868164, 0.25750935077667236, -1.3220489025115967, 0.8870612978935242, 0.9901749491691589, 0.4947011172771454, 0.22047092020511627, 0.1773741990327835, 1.2250560522079468, -0.45341789722442627, 0.7658697366714478, -0.006215389352291822, 0.2731979787349701, -0.1645621508359909, -1.4614222049713135, 1.7227871417999268, 0.17959082126617432, 0.1590558886528015, -1.3255044221878052, -0.27772197127342224, 0.28284475207328796, -0.8711877465248108, -0.7936233878135681, 0.5063572525978088, 0.5917264819145203, -1.0199202299118042, 0.6726024746894836, -0.8821712732315063, -1.5115028619766235, 0.5107381343841553, -2.089816093444824, 0.7826557755470276, -1.8727298974990845, -0.001632586121559143, -0.36851659417152405, 0.3719587028026581, 0.43365058302879333, -0.6397750377655029, 0.2164081335067749, -1.1951643228530884, 0.6240396499633789, -0.7531936168670654, 0.7775339484214783, -1.8948116302490234, -0.8860912322998047, -0.2701469361782074, 1.1168385744094849, -0.24361126124858856, 1.3312500715255737, -1.4069278240203857, 1.3831536769866943, -0.9099510312080383, 0.5857406258583069, 1.656286597251892, 0.6024953722953796, 0.6842851638793945, 0.5398894548416138, -1.481462836265564, 0.763550341129303, -1.938615322113037, -0.8213969469070435, -0.8470432758331299, 0.4416755139827728, 0.13882699608802795, 0.5740976333618164, -0.4482124447822571, -0.9948680400848389, 1.0089143514633179, -1.11069917678833, -0.34287896752357483, -1.7976638078689575, -1.8251415491104126, -0.7969317436218262, -1.4947608709335327, -1.5377637147903442, -1.5594978332519531, -0.8646650910377502, -1.2100920677185059, -1.5357508659362793, 0.22983041405677795, 0.9247242212295532, -1.4166226387023926, -1.6589624881744385, -1.7685104608535767, -0.5495559573173523, 0.3888324797153473, 1.0665513277053833, 2.198997735977173, 1.4663231372833252, 0.7887555360794067, 0.1388864815235138, -0.20529448986053467, -0.7977481484413147, 0.8431811332702637, -0.8703880310058594, 2.0440452098846436, -0.07207255810499191, -0.6588088274002075, -0.7864766716957092], [1.0297502279281616, -0.30690184235572815, 1.263581395149231, -2.649973154067993, 1.0585540533065796, 2.5762410163879395, -0.23772314190864563, 1.0623403787612915, 0.006480591371655464, 0.5630622506141663, 0.27836889028549194, 1.474621295928955, -1.0528895854949951, 0.6974648237228394, 0.01795138418674469, -1.3544548749923706, 0.09050396829843521, -0.7247222065925598, -0.7041939496994019, -0.2627622187137604, -0.07845033705234528, -0.7125886678695679, -1.5278825759887695, -1.356559157371521, -0.11552809923887253, 1.8368878364562988, -0.8339728116989136, -0.7691663503646851, -1.6323742866516113, 0.20166118443012238, 0.31473949551582336, 0.2987031042575836, -2.119948148727417, -1.0418941974639893, 0.4533363878726959, 0.8986366391181946, -0.7187134623527527, 0.8966353535652161, 0.26874199509620667, -1.0705204010009766, 1.207679033279419, 0.1234489306807518, -2.096086025238037, 1.1982649564743042, -0.205524742603302, -0.6171988844871521, 1.3362159729003906, 1.0756216049194336, 1.2907227277755737, 0.7619083523750305, 0.39278364181518555, -0.484437495470047, -1.1784470081329346, -0.0534915030002594, 1.048693060874939, 0.24792315065860748, -1.0814133882522583, 1.5557745695114136, 1.4859179258346558, 0.1414494514465332, -2.6611156463623047, -0.48399531841278076, -1.194022297859192, 0.05345407873392105, -0.16758087277412415, -0.19401049613952637, 0.5353710651397705, -0.27574265003204346, 0.9189213514328003, 0.8164797425270081, 1.2224749326705933, 2.1713509559631348, -1.8705865144729614, 0.9913860559463501, -1.1639670133590698, 1.0833443403244019, -0.6254628300666809, 1.7258400917053223, 0.6313490271568298, 1.285705804824829, -0.7943600416183472, -0.8970984816551208, -1.45078706741333, 0.6565437912940979, -0.13054189085960388, 1.1105871200561523, 0.8401727676391602, 0.06154422089457512, 2.0075247287750244, -1.145590901374817, -0.7088639140129089, 1.0606430768966675, -0.6472135782241821, 0.540597140789032, 2.1123316287994385, 0.8162712454795837, 0.09757571667432785, -0.8712019324302673, -0.6961847543716431, -0.08916369080543518, 1.236283302307129, -1.8662700653076172, 0.44024139642715454, -0.5149282217025757, 0.9446626305580139, -0.09523043781518936, -0.8109731674194336, 0.23022519052028656, -0.9413864612579346, 0.11863959580659866, -1.0435181856155396, -1.3895052671432495, -0.6734483242034912, 0.051947787404060364, 0.3303409218788147, 0.6049185395240784, -2.0458126068115234, -0.25258851051330566, 0.5165484547615051, 0.8678200244903564, -1.5911628007888794, -0.4861639738082886, -0.37664738297462463, -0.9691145420074463, 1.875953197479248, 0.7211506962776184, 0.4722164273262024, -1.8468756675720215], [0.6983855366706848, -1.773795485496521, -1.102272391319275, -0.05063871666789055, -2.1079511642456055, 0.1775081753730774, 0.3015819787979126, 0.5963051319122314, -0.34622055292129517, 0.13795918226242065, 0.22056007385253906, -0.29607802629470825, -2.2593846321105957, 0.2967822253704071, -1.9867676496505737, -0.9749146699905396, 1.3616362810134888, 0.08555857092142105, 1.6928821802139282, 2.2391109466552734, -1.462555170059204, 0.21434049308300018, 1.7587765455245972, -2.93349289894104, -1.1465529203414917, 0.10088817775249481, -0.9460089802742004, 1.4177701473236084, 0.5216280817985535, 1.3929113149642944, 0.10581545531749725, 0.8680790662765503, 2.2787413597106934, 0.397267609834671, -1.0093309879302979, 0.4340526759624481, 0.5429618954658508, 0.4879993498325348, 0.393555223941803, 0.05658577010035515, -0.15608206391334534, -0.39440515637397766, -0.022593192756175995, -0.3242345154285431, 0.579003095626831, -2.3767433166503906, 1.359476923942566, 1.2047172784805298, 0.4489535689353943, 0.7981519103050232, 0.07446049898862839, -0.11137095093727112, -1.6157876253128052, -0.8915795683860779, 1.5167536735534668, 0.41482216119766235, -0.8959594964981079, 0.029864108189940453, -2.1079585552215576, -0.4994083642959595, 0.8349084258079529, -0.6961511969566345, -3.4358737468719482, 1.3000049591064453, -0.3720097839832306, 0.9751570820808411, 0.06305947154760361, -1.3925068378448486, 0.9776986241340637, 1.772337555885315, 1.1995370388031006, 0.2762604355812073, -0.4607456624507904, -1.2603355646133423, 0.8960277438163757, 1.881536841392517, -0.15215598046779633, -0.516228973865509, 0.34396862983703613, 0.48713386058807373, 0.7081807851791382, -1.2748955488204956, -0.6735215187072754, -0.8064645528793335, 0.6209559440612793, -1.7099319696426392, -2.641761064529419, 1.4172697067260742, 1.150003433227539, -1.7608754634857178, -0.2033485323190689, -0.09349977970123291, 0.9215664267539978, -0.1835441291332245, -0.4783468246459961, -0.5530277490615845, 2.0487022399902344, -0.08447069674730301, -0.1684122383594513, 0.5357969999313354, -0.3010214865207672, 0.2623792886734009, -0.2354200780391693, -0.18250082433223724, 0.7408400774002075, -0.12435707449913025, 0.546542227268219, 0.6160657405853271, 0.1020456925034523, 0.5658623576164246, 2.86910343170166, 2.0804452896118164, 0.667079746723175, -0.030341850593686104, -0.8214719891548157, -1.4873794317245483, -0.9718638062477112, -0.7045350670814514, -0.9128066301345825, -0.707693874835968, 0.4426110088825226, -0.34053272008895874, -2.304699182510376, 1.7003724575042725, -0.4278653860092163, -0.2472468912601471, -1.1761068105697632, 0.6534037590026855], [-0.556080162525177, 0.474718302488327, -0.04199221730232239, -0.9685903787612915, 0.33776623010635376, -0.6573618650436401, 1.9013781547546387, 0.9479507803916931, 1.1738008260726929, 0.4157063961029053, 0.4049188196659088, 0.035217199474573135, 2.480201244354248, 0.823961615562439, 0.20520544052124023, 0.18313471972942352, 1.9012730121612549, -1.6368542909622192, 0.6269956231117249, 1.4069105386734009, 1.2080175876617432, -2.3228673934936523, 0.9153648614883423, 0.9360466003417969, 2.956745147705078, 1.4145017862319946, -0.7820324301719666, 0.537555456161499, 0.2522808313369751, 0.6365827322006226, 0.6531099081039429, -1.2813575267791748, -0.03236155956983566, -0.07130325585603714, 0.720162034034729, -1.5168644189834595, -0.6487494111061096, 0.571696400642395, 2.169973611831665, 1.4503076076507568, -0.19535857439041138, -1.2531198263168335, -0.388688862323761, -1.6833897829055786, -0.06596604734659195, -0.20024292171001434, 0.916441798210144, -1.1474241018295288, -0.720810055732727, -0.657457172870636, -0.773184061050415, 1.4307478666305542, 1.3003509044647217, -0.16163098812103271, -0.6642915606498718, 1.521630883216858, 1.550744891166687, -1.1190375089645386, 0.23973150551319122, -0.12649030983448029, -0.05111226439476013, -0.08748587965965271, -1.4166547060012817, -0.11745691299438477, -0.5173920392990112, 0.4587901830673218, -1.0189573764801025, -0.9576273560523987, -1.9093701839447021, 0.4981541037559509, -1.0605437755584717, -0.1831260621547699, -0.09678048640489578, 0.754146158695221, -0.31938254833221436, 0.038363151252269745, -0.2957373857498169, -0.25850483775138855, 0.06972935795783997, 0.13142867386341095, -0.6368641257286072, 0.08833593130111694, -1.0075088739395142, -0.534321129322052, 0.5889844298362732, -0.2002801150083542, -0.4107261002063751, -1.7016221284866333, 0.30869948863983154, -0.34066614508628845, 0.607323408126831, 1.1718981266021729, -1.54050874710083, -0.19881625473499298, 0.22795864939689636, -0.2658408284187317, -0.6767491698265076, 0.31082573533058167, -1.7119433879852295, -0.41921713948249817, -0.9713340401649475, -0.5420025587081909, -0.5434046387672424, 1.2813211679458618, 2.8781888484954834, 0.41858500242233276, -1.0929694175720215, -0.5606797933578491, -0.04313913732767105, -1.4719953536987305, -0.9014889597892761, 2.843994379043579, -1.016034722328186, 0.5591040849685669, 1.1049062013626099, 0.7702670693397522, -1.109093189239502, -0.5182532668113708, -0.08959459513425827, -0.762131929397583, 1.3136622905731201, 0.22608232498168945, -1.7375205755233765, 1.5797632932662964, 0.5224186778068542, -0.7296246886253357, -2.0390615463256836, -2.4123237133026123], [0.2909615635871887, 0.9338663816452026, 0.2390938550233841, -0.9075375199317932, -0.2463904619216919, -0.6664372682571411, 0.13168063759803772, 0.7827777862548828, -0.44860050082206726, 0.5328415036201477, 1.4900288581848145, -1.7684383392333984, -0.8380166292190552, -0.0015865728491917253, -1.1424777507781982, 0.12429829686880112, -0.2671244442462921, 0.43018171191215515, -1.1496646404266357, 1.177311658859253, 0.2690841853618622, -1.3438626527786255, 0.054473355412483215, 0.10268039256334305, 1.3994771242141724, 0.8117371201515198, 1.705755591392517, -0.24143408238887787, 0.15809567272663116, 0.35435813665390015, -1.1382851600646973, 0.6984377503395081, 0.41933220624923706, 0.13630647957324982, 2.0830271244049072, 0.8875256776809692, -0.17893949151039124, 1.6739709377288818, 0.17437146604061127, -0.09596692025661469, 0.4960784316062927, -0.09843531250953674, -0.8072085380554199, -0.9534953832626343, 0.6119527816772461, -0.8501517176628113, -1.528906226158142, 2.8140711784362793, 0.30050456523895264, -0.6354414224624634, 1.3382298946380615, 1.4721163511276245, -1.1629557609558105, -1.9199999570846558, 0.2533032298088074, 1.3082964420318604, -0.9491913914680481, 0.1750880628824234, 0.678078293800354, -0.5599623918533325, -1.0035361051559448, -0.5364800691604614, 1.63810133934021, -1.1047468185424805, 1.5324128866195679, -1.4823389053344727, 0.7101449370384216, -0.17634829878807068, 0.7408108711242676, 0.12945176661014557, -0.7174078226089478, -0.09532114863395691, 1.2384814023971558, 0.7304605841636658, 1.6223037242889404, 1.104284644126892, 1.8986194133758545, -0.6357982754707336, 1.2877987623214722, -0.7759032249450684, -0.5395125150680542, 1.6719247102737427, 0.024765346199274063, 1.2500369548797607, -0.24615225195884705, 0.4911757707595825, 1.337135910987854, -0.17770305275917053, 0.027274101972579956, -0.3767217695713043, -0.5898860096931458, 1.0760923624038696, 1.1202727556228638, 1.2587289810180664, 0.18223348259925842, -0.26487526297569275, -0.11209748685359955, -0.2875472605228424, 0.1860036849975586, -1.0683857202529907, -1.5728890895843506, -0.6787395477294922, -0.17955458164215088, -0.3747764229774475, 1.7225754261016846, 0.0788656547665596, 1.156704068183899, -0.9749073386192322, 2.0992727279663086, 0.3041302263736725, -0.8066221475601196, -0.013015266507863998, -1.80286705493927, -3.416686773300171, -0.3576158285140991, -0.10364347696304321, -0.9786799550056458, -0.9078721404075623, 0.8453399538993835, 1.4389582872390747, -1.739378809928894, -0.3175463080406189, -0.5646426677703857, 0.7196196913719177, -1.1565696001052856, 0.39798083901405334, 0.3188716471195221, 0.4853891134262085], [0.6431387066841125, -0.3707495927810669, 0.9335200190544128, -0.878934919834137, -0.6352190971374512, 1.0161511898040771, 0.3022862374782562, 0.4641667306423187, 0.38722050189971924, 0.41244062781333923, -0.7897436022758484, 0.8052492737770081, -0.5984559655189514, 0.170438751578331, -0.2635613679885864, 0.8888668417930603, -1.8996472358703613, -1.524544358253479, -1.3318474292755127, -0.31985363364219666, -0.6084485054016113, -0.7868567705154419, 0.33131906390190125, 0.8542018532752991, -1.0975590944290161, -0.998594343662262, 1.449332356452942, 0.7309174537658691, 0.9721726775169373, -1.0971654653549194, 1.3281745910644531, -0.8517775535583496, -0.7930541038513184, -1.27196204662323, 0.27945154905319214, 0.592043936252594, -1.184876561164856, 0.5614251494407654, 0.5766255855560303, 0.11156947165727615, 0.7443427443504333, 0.6568681597709656, 0.3693262040615082, 1.2447320222854614, 0.7151384353637695, 0.49645620584487915, -0.8861002922058105, -1.1501163244247437, -1.491695761680603, 0.7092275023460388, -0.5576045513153076, 0.08006314188241959, 0.1291281282901764, -0.9640865921974182, -2.2660579681396484, 0.14286981523036957, -0.2364169806241989, 0.9701638221740723, -0.07795284688472748, -0.042659275233745575, -1.0636638402938843, -0.7937313914299011, -0.4905537962913513, 0.12134578824043274, 0.5377913117408752, -0.1735636591911316, -0.8406689763069153, -0.19453229010105133, -1.0654067993164062, -0.11623343080282211, -0.27514412999153137, 0.4991486072540283, -0.6011613607406616, -0.2884069085121155, -0.12922486662864685, -0.5994147658348083, -0.9926010966300964, 0.6722073554992676, 1.250241994857788, -2.0675857067108154, -0.42831575870513916, 2.51216983795166, -2.1349265575408936, -0.9598783850669861, 0.5457908511161804, 3.3781166076660156, -0.365299254655838, -0.16256695985794067, -0.9805311560630798, -2.6357312202453613, -0.31788933277130127, -0.0692218691110611, -1.2346224784851074, 0.22185413539409637, -1.0742392539978027, 0.3688880205154419, -1.836182713508606, 0.07382599264383316, -1.006721019744873, 1.1527968645095825, 1.5454355478286743, -0.814437985420227, 0.27193912863731384, 0.023976150900125504, 0.6092526912689209, -1.3383270502090454, -0.09816331416368484, 1.8831337690353394, 0.6888591051101685, 0.58808833360672, 0.7229672074317932, -0.5522143840789795, 1.278527021408081, 0.6928725242614746, 0.6415296196937561, -1.3671389818191528, 0.4916543960571289, -1.604019045829773, -1.6909348964691162, -0.3103034794330597, 0.3506254255771637, 1.6188725233078003, 0.17736782133579254, -0.8636842370033264, -0.002189414110034704, -0.13742494583129883, 1.0199686288833618, -0.4493253827095032], [0.07940371334552765, 0.5501047372817993, 0.42915868759155273, 0.6563741564750671, 0.8967881202697754, -0.23925818502902985, -0.05936987325549126, 1.1200487613677979, 1.3108761310577393, 1.5300995111465454, 0.04857361689209938, 0.3427867889404297, 1.3225903511047363, -0.4450639486312866, 0.08467824757099152, -1.3351562023162842, -0.9772726893424988, -1.050162434577942, 0.28989818692207336, -1.606735110282898, 1.122772455215454, 0.2141803503036499, -0.0934562161564827, -1.1452150344848633, -1.37288236618042, -0.7872341871261597, 0.39525356888771057, -1.5035462379455566, -0.25414738059043884, 0.49643734097480774, 0.6097638607025146, 1.7699576616287231, -1.2686588764190674, -0.7018032073974609, 0.7780599594116211, -1.2349085807800293, 0.011402416974306107, -0.6731964349746704, 1.2653414011001587, -0.5732799768447876, -0.12320956587791443, 0.0747971311211586, 0.2501203417778015, -1.1668787002563477, 0.8579298853874207, 0.5928037166595459, 0.20793195068836212, 0.09449371695518494, 0.2208622395992279, 0.08342232555150986, -0.13150882720947266, 0.49127236008644104, -1.7341351509094238, 0.947941780090332, 0.3104705810546875, 0.16063600778579712, 0.4967553913593292, 0.4726274013519287, 1.8793240785598755, 0.1996065378189087, -2.0604045391082764, 0.3046261966228485, -0.5638979077339172, 0.13506947457790375, 2.0323851108551025, -1.0788335800170898, 0.8607605695724487, 0.12735968828201294, -2.684408187866211, 0.6353060603141785, 0.8835340738296509, 0.2839919328689575, 0.26314684748649597, -0.5871113538742065, -0.6804817914962769, 0.8678175806999207, -2.2178971767425537, 0.11300411820411682, -1.4452658891677856, -0.7141289114952087, 0.8760672807693481, 0.9506092071533203, -0.5589949488639832, -0.7604793310165405, -0.6824055910110474, 1.2380361557006836, -0.28826427459716797, 0.46623387932777405, 1.1209666728973389, -1.4919618368148804, 0.8030998110771179, 0.7162826061248779, 1.5892590284347534, 0.6022542715072632, -0.3041461110115051, 2.171844720840454, 1.312551498413086, -0.18560504913330078, 1.6469426155090332, -1.881137728691101, -0.9370231628417969, 0.07945969700813293, 0.20536968111991882, -0.466461181640625, 0.11957038938999176, 0.24402353167533875, 2.1759495735168457, 0.5515612959861755, 0.8984830379486084, -1.0621916055679321, -2.3886842727661133, -1.0977922677993774, 1.4131220579147339, 0.300306499004364, 1.0597050189971924, -1.6029260158538818, -1.2047796249389648, -0.29763564467430115, 0.22233782708644867, 0.5813698768615723, 0.6591854691505432, 0.1903172880411148, 0.830646812915802, 0.6454152464866638, -0.9078510403633118, -0.0011576170800253749, 0.7040822505950928, 0.347173810005188], [-0.030320681631565094, -1.7368446588516235, -1.5157238245010376, 0.6726182103157043, -0.9797017574310303, 0.3121863007545471, 1.3453670740127563, -0.14292091131210327, 1.075465440750122, -0.6883081793785095, 1.2661582231521606, -1.3636974096298218, 0.06393100321292877, -0.4976578652858734, -1.5946176052093506, 0.22158150374889374, -1.32462739944458, 0.7687506675720215, 1.2358192205429077, 0.187626451253891, -0.8107911348342896, 0.9334622025489807, 0.4809955954551697, -0.29323285818099976, -0.1648954302072525, -0.2816360890865326, -0.6798608303070068, -0.41993144154548645, 0.702294647693634, 0.204660564661026, -0.13143236935138702, 0.9794161319732666, -0.83054518699646, -0.1302415132522583, 1.558478832244873, 0.22827622294425964, -0.43975576758384705, -1.8651808500289917, 0.4046270251274109, -0.6635130643844604, -0.09016883373260498, -1.7845768928527832, 0.4962818920612335, 0.40025320649147034, -1.562178373336792, -0.5254811644554138, -1.0359641313552856, -0.49702829122543335, 0.179144486784935, 0.3687998950481415, -0.09693492203950882, -0.3733932673931122, -0.32818636298179626, 1.9551328420639038, -0.4132821261882782, -1.5814019441604614, 1.6625605821609497, 1.2629680633544922, -0.9255849123001099, 2.0249199867248535, -0.06840355694293976, -0.22013813257217407, -0.9543216228485107, -0.5300253629684448, 1.1985145807266235, -1.3754115104675293, -0.2688523828983307, -0.9380852580070496, -0.003997525665909052, -1.0639795064926147, -0.9856317043304443, -0.5925927758216858, -1.3124984502792358, 0.7158148884773254, -0.23627202212810516, 0.7003538012504578, -0.6732885837554932, 0.019145911559462547, -1.842034935951233, -0.3628681004047394, -0.4980182945728302, -0.18061463534832, 1.5322266817092896, 0.00653518782928586, -0.3405597507953644, -2.011824607849121, 0.49180033802986145, -0.9112981557846069, 1.8545745611190796, 0.013082697987556458, 1.3832483291625977, 1.5459120273590088, -0.07210691273212433, -0.935614824295044, 0.6514905095100403, 0.7444769144058228, 0.18981419503688812, -0.16530735790729523, 0.31480053067207336, 0.17063291370868683, -0.8813748359680176, -0.21988128125667572, 1.0164726972579956, -0.448254257440567, 0.4189058840274811, -0.08360730111598969, 0.22543838620185852, 1.5405353307724, 0.7396214008331299, 1.0943102836608887, -0.16310802102088928, 0.23194552958011627, 1.1764285564422607, 0.9359956383705139, 1.8113306760787964, 0.024776263162493706, 0.6424111127853394, -0.7458586096763611, 0.021207600831985474, -0.2140406370162964, 0.3189389407634735, 0.828833818435669, -1.057897925376892, 0.5831020474433899, 0.18767645955085754, -0.7877518534660339, 0.9809408187866211, -0.388657808303833], [0.2026934027671814, 0.6906387805938721, 1.9334323406219482, -0.6245592832565308, -0.765953004360199, 1.0237600803375244, 0.9855363965034485, -0.23075984418392181, -0.019038477912545204, -0.6019450426101685, -1.0411986112594604, -0.5872291326522827, 0.07743718475103378, 1.4904180765151978, 1.22101891040802, -0.25729086995124817, 1.0454678535461426, 1.5123963356018066, 0.6261261105537415, 0.34798359870910645, 1.088950276374817, 0.7134182453155518, 1.5533053874969482, -0.6224392652511597, 1.087160348892212, -0.9798054099082947, 0.56231689453125, 0.6477251648902893, 0.8880340456962585, -0.02316613867878914, 8.346988761331886e-05, 0.07990951091051102, -0.6173614859580994, -0.06549955904483795, -0.7789402604103088, -0.3634085953235626, 1.2224947214126587, 0.4318332374095917, -0.8950092196464539, 0.1503204107284546, 1.1475645303726196, -0.2637200951576233, -1.9443825483322144, -1.5923118591308594, -0.8401973843574524, 0.09468140453100204, 0.7661020755767822, -1.5182138681411743, -0.5225584506988525, 0.041127149015665054, 1.6964558362960815, 2.0274441242218018, 0.8587547540664673, 1.5903979539871216, 0.6315611004829407, 0.09298841655254364, -0.455181360244751, -0.8051801323890686, 0.06489715725183487, -0.15058469772338867, 0.25367897748947144, 1.2024815082550049, -0.819699764251709, 0.33183395862579346, 1.7852667570114136, 0.3243338167667389, 1.4803552627563477, -0.5564469695091248, -2.91978120803833, 0.48138776421546936, 0.22272545099258423, 1.1971107721328735, -0.7487099170684814, 0.2330574244260788, 0.06779198348522186, -0.549338161945343, -0.3166784346103668, -0.2565465569496155, -0.469649076461792, -0.7412768006324768, 0.08387783914804459, 0.14627531170845032, -0.6692228317260742, -2.1831929683685303, -0.21963736414909363, -0.9407050609588623, -0.7708550095558167, 0.4683423638343811, 1.0750819444656372, -0.18612100183963776, 1.2818291187286377, -0.9631600379943848, -3.1114540100097656, 1.1204105615615845, 1.1891180276870728, 0.3017059862613678, 0.38035500049591064, 1.6202069520950317, -1.5041717290878296, 0.377308189868927, 0.179209366440773, -0.5592906475067139, 1.6724610328674316, -1.1949752569198608, -1.5528230667114258, -0.7454426288604736, 0.28528082370758057, -0.5309140086174011, 0.4835749864578247, -0.7778880596160889, -0.5856271386146545, -0.2136060893535614, -0.9123784899711609, 1.292798638343811, -1.5123605728149414, 1.1323012113571167, 0.2209181785583496, 2.526655673980713, -0.7666530013084412, 1.2873705625534058, -1.2781362533569336, 0.1974315345287323, -1.9501290321350098, 1.5778818130493164, 0.974761962890625, -0.641940176486969, -0.5425825715065002, -0.43676674365997314], [0.31880319118499756, -0.34970149397850037, -0.3049179017543793, -0.4322573244571686, 0.748021125793457, -1.2162257432937622, 1.9042738676071167, 1.3384733200073242, 1.8511736392974854, 0.45549312233924866, 0.11799976974725723, -0.19235335290431976, -0.8695381879806519, -0.9988535046577454, 0.19235116243362427, -0.4204532504081726, 1.8601274490356445, 0.8653858304023743, -0.4703249931335449, 0.7978596091270447, 1.4643150568008423, 1.124879240989685, 1.125731348991394, 0.5862370133399963, 0.4903525114059448, 1.810503363609314, 0.7101044058799744, 1.7296241521835327, 0.9491735100746155, -1.0588698387145996, 1.0408210754394531, 1.6395336389541626, 0.16393204033374786, -1.8351504802703857, 1.4956053495407104, -0.5929845571517944, -1.5013257265090942, -1.5627986192703247, 0.12418168038129807, 0.5581939816474915, -0.923209011554718, 0.09469932317733765, 0.09945198893547058, 1.6981403827667236, 0.4744773209095001, 0.6891348958015442, 1.1688381433486938, -0.773389995098114, 0.24199353158473969, -1.740695595741272, 0.06322336196899414, 1.342268943786621, -2.070953607559204, -0.3221340775489807, 0.8770272731781006, 1.886448860168457, 0.9437713027000427, -0.6312077641487122, 0.8579921722412109, 1.2737596035003662, 0.7699694633483887, 1.522127389907837, 2.9073548316955566, -0.30179524421691895, 0.7085548639297485, -1.0440504550933838, 0.39742016792297363, -0.8785706758499146, -0.24817794561386108, -1.0560634136199951, 2.0476009845733643, 1.7458462715148926, 1.8356947898864746, 0.2349967062473297, 0.49840590357780457, -1.3232537508010864, 1.3675576448440552, -0.897579550743103, -0.45340868830680847, 0.5404161810874939, -0.3311145305633545, 0.4561440944671631, 0.6893104910850525, 0.5545542240142822, -0.6757873892784119, -0.6000633239746094, -0.8898217678070068, -2.548891305923462, -0.5333441495895386, -0.6128318905830383, 1.138569951057434, -0.03510013967752457, -0.8934636116027832, 0.26824647188186646, -1.4648218154907227, -0.4542985260486603, 1.1582988500595093, -1.0750333070755005, -0.5107864141464233, 0.20457352697849274, 0.05428025871515274, 1.473671555519104, 0.23392826318740845, -0.43647822737693787, 1.0228568315505981, -0.17647920548915863, 0.059658680111169815, 0.1644916981458664, 0.22554011642932892, -1.2702304124832153, 0.5656086802482605, 0.25723811984062195, 0.806877851486206, -1.738316535949707, 0.5476157069206238, -0.09103307127952576, -1.3357361555099487, -2.450364351272583, 0.35575568675994873, 1.3900463581085205, -0.9235349297523499, 0.5137325525283813, -0.43762198090553284, 0.7821213006973267, 0.5826863646507263, -1.277435302734375, 0.7406367063522339, 0.03354497253894806], [0.6670389771461487, 0.26689985394477844, 0.9683048725128174, 1.0854192972183228, -1.3681583404541016, -0.1698121428489685, -0.7620118260383606, 1.5774030685424805, -0.5847107172012329, -2.17147159576416, -0.7300528883934021, 0.27866560220718384, -1.562053918838501, 2.6724860668182373, 0.4048272967338562, -0.7138912081718445, 0.7278752326965332, -0.7118419408798218, 0.29632502794265747, -0.23418255150318146, -0.5457959175109863, -0.9969808459281921, -0.4037201404571533, 0.9790439009666443, -1.1533498764038086, 0.17839276790618896, -0.42005106806755066, -0.6349347233772278, 0.948842465877533, -0.4861529767513275, 0.3713986277580261, 0.42702892422676086, -1.13490891456604, 0.17808088660240173, 0.574589192867279, 0.4871894121170044, -1.2230169773101807, 0.37483832240104675, -3.4190967082977295, -2.2241411209106445, -0.6624370813369751, 0.27167338132858276, -1.5823696851730347, -1.7906029224395752, -0.8195711970329285, 0.4029610753059387, -0.16662082076072693, -0.03666062653064728, -0.16013844311237335, 0.4068696200847626, 0.011479564942419529, 1.1908968687057495, 1.1621443033218384, 0.09249398857355118, 1.2310206890106201, -0.16161994636058807, -1.1619701385498047, 0.49822187423706055, -0.9413446187973022, -0.06725894659757614, 0.5596003532409668, -0.7575998306274414, -0.3343774974346161, 0.5880276560783386, -0.3645079433917999, -1.9181627035140991, -0.7589508891105652, -2.3316075801849365, 0.8477463722229004, 0.13270312547683716, -0.6472005844116211, -0.738431453704834, -0.13648682832717896, 1.5713708400726318, 0.01172194629907608, 1.2654715776443481, -0.7898964881896973, -0.6821109056472778, -0.09995269775390625, 1.4856306314468384, 1.7231594324111938, 0.023226531222462654, 0.04526253044605255, -0.3615024983882904, -1.4287163019180298, -0.10086707770824432, -2.2153923511505127, 1.766038179397583, -0.1458812952041626, -1.481680154800415, 2.3346025943756104, 0.5360125303268433, 0.5311242341995239, -1.7344411611557007, -0.428772509098053, -0.91136234998703, 0.32772883772850037, 0.7943174242973328, -0.4266098439693451, 0.411933571100235, -0.6788775324821472, 0.14614185690879822, 0.08593904227018356, 0.37894392013549805, -0.7692762017250061, -0.08969101309776306, 0.03402415290474892, 0.8902423977851868, 0.2971270978450775, 0.9898414015769958, 0.8124328851699829, 1.7139968872070312, -1.9185798168182373, -0.2733319401741028, -0.08425384759902954, 0.4310459792613983, 0.4208334982395172, -0.9179670214653015, 0.13721896708011627, 2.5857176780700684, 1.0764909982681274, -1.358414888381958, 1.1344603300094604, 0.06942200660705566, -0.7080109119415283, -1.5244190692901611, -1.4275263547897339, 0.41105401515960693]], [[0.11738217622041702, -0.28452953696250916, -0.25643494725227356, 0.37069588899612427, 1.6335737705230713, -0.5187612771987915, 0.2988796830177307, -0.5340548753738403, -0.2008253037929535, 1.8774808645248413, -0.8917354941368103, 0.23413221538066864, -1.9321234226226807, 0.6452655792236328, 0.24333685636520386, -0.29961448907852173, -0.3674311637878418, -1.138014554977417, -0.1782420128583908, 1.1606149673461914, -1.2056217193603516, 1.0264537334442139, -1.3774532079696655, -1.8893826007843018, 0.14627686142921448, 1.0425493717193604, 1.576013445854187, -0.0832352340221405, 0.00246595311909914, 0.32766440510749817, 0.416951060295105, -1.2980382442474365, 1.4567298889160156, 0.3740122318267822, -0.9826667904853821, -1.9194321632385254, -1.6876769065856934, 0.4287025034427643, -0.04970988258719444, -0.02465488202869892, -0.037669677287340164, -0.28155240416526794, -1.4165699481964111, -1.0191274881362915, 0.017908789217472076, -1.0995454788208008, 0.7895089983940125, 0.6463949084281921, 0.07365212589502335, -0.29885855317115784, 0.7649855017662048, 0.856507420539856, -0.7094451189041138, -0.8241392374038696, -1.427182912826538, 0.007819618098437786, -0.2538403868675232, 0.6655129194259644, 0.34814754128456116, -0.9935615062713623, 0.3139974772930145, -1.795021891593933, 0.7140526175498962, 0.3392705023288727, -1.520817756652832, 0.06171228364109993, 1.2975547313690186, 1.310370683670044, 0.7829875349998474, -0.10614456981420517, 0.44795477390289307, -0.3788999021053314, 0.668707549571991, -0.044586360454559326, -1.2981454133987427, 0.20257531106472015, -1.366077184677124, 0.49873194098472595, 1.7038087844848633, -1.688527226448059, 2.0363657474517822, -0.7406644821166992, -1.0354053974151611, 0.6859853863716125, -0.08798400312662125, 0.6057551503181458, -1.2408843040466309, -1.7349132299423218, 1.0786969661712646, 0.030380740761756897, 0.2174028605222702, -0.9141533970832825, -0.37874436378479004, -1.6544179916381836, 0.8739286065101624, 0.22148160636425018, 0.5094283223152161, 0.39715293049812317, 2.1369869709014893, -0.7673510909080505, 0.37571442127227783, -0.23457267880439758, 1.0256106853485107, 0.9683157205581665, 0.03175411745905876, 0.29461240768432617, -0.4569430947303772, -0.8218176364898682, -0.15274739265441895, -0.08954036980867386, 0.19819721579551697, 0.5317929983139038, -0.8963575959205627, 1.5191985368728638, -0.6911364197731018, -0.733609676361084, -0.412244588136673, 0.516074538230896, 0.035421065986156464, 0.07744606584310532, 0.9254966378211975, -0.5411695837974548, -0.8171749711036682, 2.1944100856781006, 0.09606817364692688, -0.748985230922699, -0.5333720445632935, 0.5095811486244202], [0.5656719207763672, 1.6501976251602173, -0.1757407784461975, 0.4081610441207886, 0.7844440937042236, -0.6833357810974121, -0.7051281929016113, 0.722196638584137, 1.0200064182281494, 2.444640874862671, -2.0326952934265137, -0.7090276479721069, -0.8083912134170532, 1.1251733303070068, 0.8760108947753906, -0.2177010327577591, -1.9365495443344116, -1.302026391029358, 1.2539936304092407, 1.4385428428649902, -0.4304104149341583, 0.545994758605957, 0.05124916508793831, 0.2741544246673584, 0.7591817378997803, -0.5266432762145996, 1.3850104808807373, -0.06807132810354233, -0.7562919855117798, 0.21756881475448608, -0.12577655911445618, 1.8663586378097534, 1.690227746963501, -0.7086697220802307, 2.281543016433716, 2.0854744911193848, -0.48010188341140747, 1.7939722537994385, 0.2925475537776947, -0.753116250038147, 1.0300017595291138, 1.2481446266174316, 0.5335443019866943, -0.4187106490135193, -0.10110097378492355, 0.2863514721393585, -1.0750643014907837, -0.346331387758255, -0.14480489492416382, -1.5974231958389282, -0.11814795434474945, 2.126037120819092, 0.01377064362168312, 0.004799245856702328, -0.13168717920780182, -0.5003218054771423, 1.0701465606689453, -1.179400086402893, 0.19735516607761383, 0.0815582424402237, -1.5768582820892334, 1.0385534763336182, 0.01827976480126381, 0.16614699363708496, -1.0330089330673218, -1.1730601787567139, 0.7753342390060425, -0.016531523317098618, -0.48754018545150757, 0.23441003262996674, -0.027316056191921234, -0.058038145303726196, 0.10607326775789261, -0.7945929169654846, 1.4337106943130493, -0.9921513199806213, -1.4046343564987183, 0.10562087595462799, -0.4520806670188904, 0.4414496123790741, -0.7160648703575134, 0.8456874489784241, -0.8496333956718445, 1.1396374702453613, -0.559646725654602, -0.7667518854141235, -1.3308167457580566, -2.038550853729248, -0.140292227268219, -2.9629483222961426, 0.757599413394928, -0.9254327416419983, -2.4456822872161865, 1.0161019563674927, -0.8277366757392883, -0.9545183777809143, -1.010021448135376, -0.23986537754535675, -0.31672006845474243, -1.2015496492385864, -0.207622230052948, 0.6612286567687988, 1.3401298522949219, 1.358294129371643, 0.9107419848442078, -0.032967645674943924, -0.5812108516693115, 1.0805182456970215, 0.33343103528022766, 0.8406151533126831, 0.49143460392951965, 1.504483699798584, 0.9855679273605347, 1.0109671354293823, -0.22520127892494202, 0.2881968915462494, -0.21273182332515717, 0.16332510113716125, -1.8889254331588745, 0.3814254403114319, -1.7148383855819702, -1.667173147201538, -0.14219728112220764, -2.237032413482666, -0.9408167004585266, 1.416420817375183, 0.2737780809402466, 0.9731608629226685], [-0.12227120995521545, 0.10247616469860077, 1.1551049947738647, -2.8088579177856445, -0.8776087760925293, 0.20628361403942108, -0.6073667407035828, -0.16705437004566193, 1.4473975896835327, -0.8009686470031738, -1.9895704984664917, -0.9848519563674927, 1.6978017091751099, 0.21544940769672394, 0.3173368573188782, 0.48013824224472046, -1.5940091609954834, 0.2115171104669571, 0.4807562828063965, -1.9724061489105225, 1.7572578191757202, -0.3377498388290405, 0.14237312972545624, -0.2682439684867859, -0.8179821968078613, 1.3060811758041382, -0.8793309926986694, -0.4655747413635254, 1.9623388051986694, 0.11024352163076401, -1.684876561164856, -0.35553476214408875, -0.5310198068618774, 1.601638674736023, -1.57395601272583, 0.08040733635425568, 1.5547094345092773, 0.2568247318267822, 0.003082594834268093, -0.8950788378715515, -0.583434522151947, -0.5251625180244446, 0.38106897473335266, -0.1994408518075943, 1.207121729850769, 0.034382760524749756, -0.08097195625305176, -0.8541483879089355, -0.8828360438346863, 1.0317951440811157, 0.7623845934867859, -1.376513123512268, -0.10308366268873215, 1.0692729949951172, -0.5872623920440674, 0.749726414680481, 0.9992414116859436, -0.22358787059783936, -0.5121302604675293, 0.6003074049949646, -0.8390719294548035, -0.34433451294898987, -0.2175855189561844, -0.9009233713150024, -0.020737409591674805, -1.190327525138855, 0.27914413809776306, -0.2887115180492401, -0.8680531978607178, -0.5509130954742432, 0.5358132719993591, -0.971327006816864, -0.37290510535240173, -0.1918969452381134, -1.2457034587860107, -0.6499612927436829, 1.2865480184555054, -0.0037632633466273546, 0.731990396976471, 2.335529088973999, 0.7072285413742065, 0.6504451036453247, -1.3369024991989136, 0.10352376103401184, -1.2619599103927612, -0.16310174763202667, -2.824918508529663, 0.09462273120880127, 0.18713189661502838, 1.1391551494598389, -0.011449221521615982, 0.055342938750982285, -0.1542913019657135, -1.0624200105667114, 0.012394889257848263, -1.634657382965088, -1.0792722702026367, -0.6374289393424988, 0.6002832651138306, -0.3210045099258423, -1.8335798978805542, -0.07153313606977463, 0.6148070693016052, -0.42840415239334106, -0.07443629950284958, 0.03726877644658089, 0.015100711956620216, -1.3904739618301392, -0.030019745230674744, 0.4096629321575165, 0.2704857885837555, 1.235691785812378, -0.20339521765708923, -1.1310237646102905, -1.394579291343689, 1.6108320951461792, 0.18499256670475006, 0.6385555267333984, 0.472059041261673, -0.06601817905902863, 0.053503263741731644, -0.043941181153059006, -0.050608936697244644, 1.457341194152832, -0.08503947407007217, 0.3596336841583252, 0.05303594097495079, 0.7733772397041321], [0.08488783985376358, -1.671923279762268, -1.264031171798706, 0.3888055384159088, 0.21397463977336884, 1.6197192668914795, -0.6311836242675781, -1.891005039215088, 0.06593940407037735, 0.46814045310020447, 1.3403347730636597, -0.5579165816307068, 0.05621535703539848, 0.340409517288208, -2.416616201400757, 1.8830434083938599, -0.05162004008889198, 0.1912783980369568, 1.5541001558303833, 0.8374976515769958, 1.1574373245239258, 1.0884618759155273, 1.3715919256210327, -1.0908658504486084, 1.0153683423995972, 1.3743765354156494, 0.3783911466598511, 0.6539704203605652, 0.532383382320404, 0.5771792531013489, -0.3925626277923584, -1.1566377878189087, -0.31759700179100037, -1.9035512208938599, 1.8116618394851685, 0.5146845579147339, 1.452649474143982, -0.6029017567634583, 1.3806872367858887, -0.14835669100284576, -1.013739824295044, 1.7862643003463745, 0.0627698004245758, 1.9120862483978271, 0.3362976014614105, -1.2712122201919556, -0.23498667776584625, 0.10145340114831924, 1.0471879243850708, -0.31644728779792786, 0.28937438130378723, 0.9189870953559875, 2.154754877090454, -0.05869916453957558, 0.3019336462020874, -0.8014488220214844, 0.3496197760105133, -0.22424441576004028, -2.2824039459228516, -0.06318660080432892, -2.4534268379211426, -1.152297854423523, -0.09155379235744476, 1.4107099771499634, 0.5386040806770325, 1.5208526849746704, -0.06072114035487175, -0.9797869324684143, -0.3929639160633087, -1.5228320360183716, 1.3205398321151733, 0.7402842044830322, 1.941508173942566, 0.2288946658372879, -0.037915993481874466, -0.6740268468856812, -0.665144681930542, 1.1990565061569214, -0.14079806208610535, 1.8238108158111572, 0.4927183985710144, -0.7510911822319031, 1.0525474548339844, -1.71793532371521, 1.2948657274246216, -0.5900337100028992, 0.8410466909408569, -0.7292947769165039, 0.17721576988697052, 0.1447737216949463, 0.6565196514129639, -0.11038349568843842, 1.690353512763977, 0.868668258190155, 0.38317927718162537, 0.055595315992832184, 1.0225389003753662, -1.59038507938385, 0.7363277077674866, -2.354454755783081, 1.4521771669387817, -0.3446812033653259, -0.20261165499687195, 0.17157818377017975, -0.07639133930206299, 0.9802597165107727, -0.630741536617279, -0.2525309920310974, -1.0146806240081787, -0.034031495451927185, 1.240903615951538, -1.2900069952011108, -0.5755882263183594, -1.4500327110290527, 0.8458544015884399, 1.7137974500656128, 1.487034797668457, 0.3911077380180359, 2.4433062076568604, 0.2049500197172165, -1.9923368692398071, 1.4797908067703247, 0.7531202435493469, -0.4817923903465271, 0.7266111969947815, -1.655877709388733, -0.707034170627594, -1.006754994392395], [1.6728311777114868, 1.4485539197921753, -2.5585720539093018, -0.8943719267845154, -1.3323370218276978, -1.1393176317214966, -2.1548874378204346, 1.5198595523834229, 0.8894383907318115, -0.583309531211853, 0.27464962005615234, 0.1276487559080124, 0.911742627620697, -0.4038734436035156, 0.3619498610496521, 1.79379403591156, 1.6069973707199097, -0.7759314179420471, -0.4024173617362976, -2.094141721725464, -1.171393871307373, -0.4171517491340637, 0.9540921449661255, 0.8796599507331848, 1.5211228132247925, 0.5532175302505493, 2.3956894874572754, 1.5369259119033813, -0.630593478679657, 1.2214213609695435, -0.9271698594093323, -0.1411532610654831, -0.7574848532676697, -0.313067227602005, -0.9467447996139526, -0.46485579013824463, 0.14728175103664398, -0.5615279674530029, -0.5966064929962158, 0.4574776589870453, 0.20883171260356903, 0.7290701866149902, -0.4624782204627991, -1.6392316818237305, -1.1811848878860474, 0.563589870929718, 1.1321749687194824, 1.3541163206100464, 0.9447062611579895, 0.40549641847610474, -0.5893660187721252, 0.6243466138839722, -0.8748548626899719, -1.1579279899597168, 0.10878255218267441, 1.7618111371994019, -1.110369324684143, -0.1366344839334488, 0.8745839595794678, 0.5735336542129517, -0.30615338683128357, 1.150228500366211, -0.9375376105308533, -2.002946138381958, -0.45123574137687683, -0.7066085338592529, -1.0225235223770142, -1.081792950630188, 0.7576206922531128, 0.5648878216743469, -0.6309213042259216, 1.5253583192825317, -1.0644842386245728, -1.8649202585220337, -0.13375456631183624, -0.8315325379371643, 1.337327480316162, -0.35677632689476013, -0.9712457060813904, -1.3181755542755127, -0.18501193821430206, -1.1614127159118652, 0.6204254031181335, 1.4749462604522705, -0.35963770747184753, 0.185737743973732, -1.4745603799819946, -1.2125736474990845, -1.8428144454956055, -1.3603167533874512, 0.7898721098899841, 0.32967865467071533, -1.5188144445419312, -1.0280689001083374, -0.2354988157749176, -1.1791043281555176, -0.4556503891944885, 0.9308029413223267, 1.6570178270339966, 0.562389075756073, 0.1929066777229309, -0.8004661202430725, -0.2879619598388672, -0.24900591373443604, -0.21278312802314758, 0.43085047602653503, 1.096479892730713, -0.03846515342593193, 1.20195472240448, -1.0601563453674316, -1.3446825742721558, 0.6895497441291809, -1.3069056272506714, 0.707452118396759, 1.3067742586135864, -0.7551349997520447, -1.0457160472869873, -1.3542869091033936, 0.05328156426548958, -1.0002621412277222, -0.11236759275197983, -0.6243605613708496, 1.074263334274292, -0.636076807975769, 1.7093682289123535, 0.19215108454227448, -0.5341366529464722, 1.623855710029602], [-0.5389488339424133, 1.084226369857788, -0.22095736861228943, 0.17555032670497894, -0.34093326330184937, 0.4755953252315521, 0.7758156061172485, 0.4350847005844116, -1.4473354816436768, 1.7822911739349365, 0.09011869877576828, -1.208999514579773, 1.871212363243103, 0.4005013406276703, -0.12765154242515564, -0.0677853375673294, -1.6286572217941284, -1.0161454677581787, 0.5203171372413635, 0.5684651732444763, 0.5144355893135071, 0.5144380331039429, -0.570722758769989, -1.13953697681427, -0.45027846097946167, -0.8816969394683838, -1.9530694484710693, 0.0743374228477478, -1.6052401065826416, 0.03288300707936287, 0.978644073009491, -1.2490215301513672, -0.7704952955245972, -0.6093847751617432, 0.03649487346410751, -0.6688988208770752, 0.14377403259277344, -0.26684075593948364, 2.179309606552124, 0.2452721744775772, -0.32271409034729004, 0.8304451107978821, -0.37053799629211426, -0.5734895467758179, -0.9820836782455444, -0.8156343102455139, 0.12086109817028046, 1.3981884717941284, -2.1581714153289795, 1.3459010124206543, -0.23960046470165253, 0.13219515979290009, 1.52415931224823, -0.8665053844451904, -0.08127229660749435, -0.22546924650669098, 1.2882118225097656, -0.08281490951776505, 1.1054962873458862, -0.4640292823314667, -0.1569146066904068, -0.1027851477265358, -2.1457653045654297, 0.5597124099731445, 0.5291639566421509, 0.3244161605834961, -1.9434705972671509, 0.28115957975387573, -0.17187434434890747, 1.5709025859832764, 0.9686650037765503, 0.5530759692192078, -0.1292959600687027, 1.506909728050232, -0.9729659557342529, 1.7072237730026245, -1.5065714120864868, 3.110316753387451, -0.9176006317138672, 0.872263491153717, 0.6756049394607544, -1.517109751701355, -0.3503677248954773, -1.8064802885055542, -0.8876100778579712, -0.5821583867073059, 1.1197766065597534, 1.077872395515442, 0.9930815696716309, -0.3294685184955597, -0.6860388517379761, -0.7584745287895203, -1.3617651462554932, 0.8160693645477295, 0.7210496068000793, -0.6761510968208313, 1.2420729398727417, -0.3064934313297272, -0.6144939661026001, 0.16591022908687592, -0.6331456899642944, 1.018009066581726, -0.8358727693557739, 0.9096275568008423, -0.10009367018938065, 0.3344021439552307, 0.058034852147102356, 0.38098645210266113, 0.1000916063785553, 1.2189940214157104, 0.06935745477676392, -1.0152199268341064, 1.4963889122009277, 1.307927131652832, -1.2048548460006714, -0.2287120223045349, 1.4149049520492554, -0.2205180525779724, 1.4790935516357422, -0.8065246343612671, -0.8209725618362427, 0.0935782864689827, 0.1826271265745163, -0.48125937581062317, 1.2687592506408691, -0.2292400449514389, -0.42808303236961365, -0.990339457988739], [0.13423970341682434, 0.9375690221786499, 0.5742986798286438, -1.4145798683166504, -0.11733872443437576, 0.513627290725708, 2.0026278495788574, 1.3380632400512695, 1.7894682884216309, -1.8227168321609497, 0.26582446694374084, -0.362657755613327, -2.157109260559082, -0.1192602664232254, 0.08711054921150208, 1.2727431058883667, 0.6732532978057861, -0.18535414338111877, -1.0142070055007935, -0.4806787073612213, 0.7118718028068542, -0.8353721499443054, 0.17703087627887726, -2.2894914150238037, 0.2897183895111084, 1.869491457939148, 0.13812631368637085, -1.536918044090271, 0.46912139654159546, 0.3796493113040924, 1.0485544204711914, 1.9527448415756226, -1.311347246170044, -2.0430760383605957, 0.9485774636268616, -0.013228017836809158, -0.22973844408988953, 0.7768288254737854, -0.027193473652005196, 0.9622337818145752, 0.9626157879829407, -0.9544321894645691, -1.1042134761810303, -0.23700769245624542, 1.6021075248718262, 0.13208164274692535, -1.067550539970398, -1.6129813194274902, -1.1907505989074707, -0.002572101540863514, -0.43924233317375183, -0.02397337183356285, 0.12630559504032135, -0.1354232132434845, 0.5361692905426025, -0.40819820761680603, 0.2703537344932556, -0.16253389418125153, 0.21615993976593018, 0.9540842175483704, -0.36728736758232117, -1.6483311653137207, -0.12704089283943176, -0.18862873315811157, -0.0015564421191811562, 0.7944839000701904, 1.5501521825790405, 1.681523084640503, -0.7365191578865051, -0.7871273159980774, 0.17932680249214172, 1.2932389974594116, 1.1282870769500732, -0.19779878854751587, 0.35862743854522705, -0.9442029595375061, -0.3807515799999237, -2.1046950817108154, 0.8093056082725525, 0.8036876320838928, -0.5208267569541931, -0.428585410118103, 0.8727707266807556, -1.22637939453125, -0.17267674207687378, 0.0012426787288859487, -0.3587264120578766, -0.7316232919692993, 1.6983805894851685, -0.06437553465366364, -1.323528528213501, 2.4153947830200195, -0.21264363825321198, -1.6925718784332275, 0.5102208852767944, -0.5550726056098938, 1.7813369035720825, -0.8588209748268127, -2.6363606452941895, -0.7617028951644897, 1.0131453275680542, -1.2609848976135254, -0.1583770364522934, 1.181359887123108, -0.6719173192977905, -1.0370150804519653, -0.4001528024673462, 2.2153546810150146, 0.736062228679657, -1.3003901243209839, -1.2110508680343628, 0.15759223699569702, 1.147420883178711, -1.302887201309204, 0.3324405550956726, -1.0647326707839966, 0.2699233293533325, 0.6691559553146362, -0.497639000415802, -0.001792146940715611, -0.5600998997688293, -2.6622819900512695, -0.18572813272476196, -0.5134543776512146, -0.3565114140510559, -0.44580093026161194, -1.09366774559021, -0.5679590106010437], [0.18803349137306213, -0.9893667697906494, 2.2394373416900635, -0.4622330665588379, 1.3018237352371216, 0.7223829030990601, 0.018942462280392647, -0.18831001222133636, -0.8246868848800659, -1.0795999765396118, -0.645480751991272, 0.6068132519721985, 0.8430998921394348, 0.596074104309082, -0.3521166741847992, 1.174879789352417, 0.4009401798248291, 0.7798557281494141, 0.20918555557727814, 1.2526023387908936, 0.5471222996711731, 1.9558677673339844, 1.2031104564666748, 0.9753023386001587, -0.6782723069190979, -1.1542778015136719, 0.43057510256767273, -0.9959401488304138, 0.9135249853134155, 1.6922844648361206, 0.6460492014884949, 0.3493247628211975, 0.9951351284980774, 0.06549303233623505, -0.1989934891462326, 0.08138392865657806, -0.18592464923858643, -1.6689634323120117, -1.2669346332550049, -0.896876335144043, 1.0808024406433105, 0.1560182422399521, -0.18202117085456848, 0.5967822074890137, 0.323372483253479, 0.883151113986969, -0.1332607865333557, -1.9086474180221558, 1.0505019426345825, -1.2298989295959473, -0.4644121527671814, 0.43619465827941895, -0.01571039855480194, -0.14769355952739716, -0.4023490846157074, -0.49532556533813477, -0.12638939917087555, 0.7956096529960632, -1.3326654434204102, -0.3335489332675934, -0.34465351700782776, 0.932630181312561, 0.6833726763725281, 0.40191325545310974, 0.5318313837051392, -0.8237621188163757, -1.396794080734253, 0.40747055411338806, -1.1356279850006104, 1.0349339246749878, 0.76573646068573, 1.3267285823822021, -1.1747009754180908, -0.7662330269813538, 2.1803860664367676, 0.6372933387756348, 1.159605622291565, -0.12547393143177032, 1.9230859279632568, 0.18985608220100403, -0.5221319198608398, 0.25871965289115906, -0.20963898301124573, -0.9667688012123108, 0.6375941038131714, 2.5943806171417236, 2.323171615600586, 0.02463284507393837, -0.866558849811554, 1.2082910537719727, 0.2724888026714325, -0.6042906641960144, -0.6392096877098083, -0.6007975935935974, -0.23773297667503357, 0.6801725625991821, 0.38149744272232056, -0.5012771487236023, -0.5531905293464661, 0.8152554035186768, -0.16799229383468628, 0.43107321858406067, -0.8002694249153137, 0.05916250869631767, 0.35902711749076843, 0.10928601026535034, -0.8583585023880005, -1.208380103111267, 0.4336095452308655, 0.4817436635494232, 2.4658560752868652, -0.6653177738189697, 1.0632132291793823, 0.26574862003326416, 1.3683170080184937, -1.1748994588851929, -1.6272720098495483, -1.6340123414993286, -1.9418003559112549, 1.1482698917388916, 0.3726647198200226, -0.8816478252410889, 0.8785611391067505, 1.4451446533203125, -0.8947716355323792, -0.7519158720970154, 0.12176033109426498, -0.6500492095947266], [-0.24807369709014893, 0.1963277906179428, -1.3804823160171509, -1.0520933866500854, -0.5241790413856506, -1.5840294361114502, 0.6613433957099915, 1.883054256439209, -0.2776649594306946, 0.8620990514755249, -0.0047975946217775345, 0.6358075737953186, -1.2629767656326294, 1.2155548334121704, 1.8029729127883911, 0.6792271733283997, 0.07774252444505692, -0.34108594059944153, -0.9384840726852417, 0.07402929663658142, -1.2682775259017944, -0.31119996309280396, 0.8438705801963806, -0.974706768989563, -0.5963652729988098, 1.9597615003585815, -0.15161184966564178, 0.7014375329017639, 1.3567395210266113, 0.11190146207809448, -0.8105193972587585, -0.41919511556625366, -1.1422525644302368, 0.45181804895401, -0.8539207577705383, 0.22902046144008636, -0.5980926752090454, -0.04599585384130478, 0.6405586004257202, 0.681415319442749, 1.4829308986663818, 0.8289811611175537, -2.1237223148345947, 0.35775884985923767, 0.8784214854240417, -1.3721818923950195, -0.0019557105842977762, 0.16400350630283356, -0.6762923002243042, 1.1182719469070435, 0.6321773529052734, -0.08733757585287094, 1.2486480474472046, 0.5478620529174805, -0.2502143979072571, 1.0275499820709229, 1.7828025817871094, -0.3986884355545044, 0.17685841023921967, 1.658418893814087, -0.10271111130714417, -0.036933064460754395, -1.540755033493042, -1.485917091369629, 0.41624313592910767, 1.6198936700820923, 0.16582736372947693, -0.03879847377538681, -1.697489857673645, 0.6724876165390015, 0.6289288401603699, -1.9019590616226196, -0.46088460087776184, 1.461295485496521, -0.7664908170700073, -1.28816819190979, 1.5445009469985962, 0.1186429113149643, -0.4919583201408386, -1.5491039752960205, -0.4446871280670166, 0.25283220410346985, 0.5314562916755676, -0.24202102422714233, 1.5251108407974243, 0.7557932138442993, 1.3806204795837402, 0.031309355050325394, 0.1951112300157547, 0.15719106793403625, 1.8133199214935303, -0.43129459023475647, 0.384440541267395, 1.0654264688491821, 0.6637437343597412, -0.27608522772789, -0.7870912551879883, 1.862007737159729, 1.5465424060821533, 0.06785798072814941, 1.6749171018600464, 0.5461854934692383, -1.1082788705825806, -0.961549699306488, 0.45461249351501465, -0.7946377992630005, -0.5556106567382812, 0.7776158452033997, -0.7681475281715393, -1.8683274984359741, -0.28423938155174255, 1.6057429313659668, -0.12474536150693893, -0.9645246267318726, -0.20554983615875244, 1.8669984340667725, 0.3697311580181122, -0.9841843247413635, -0.8555154204368591, 0.9285097718238831, 0.666160523891449, 0.08363570272922516, -0.8048239946365356, -0.5134908556938171, 0.6847864389419556, 0.9998024106025696, -1.049951195716858, -0.7598459720611572], [0.6656841039657593, 0.3694629967212677, 1.3181675672531128, -0.0030020023696124554, 0.053227927535772324, 0.1590927243232727, 1.1543198823928833, -0.6171562671661377, 0.5115481019020081, -0.39673691987991333, 0.0030374382622539997, 0.23306696116924286, 0.908928394317627, -3.1562271118164062, -0.12765707075595856, 0.1124863252043724, 0.18585900962352753, 0.934468686580658, 0.44398993253707886, 0.33664172887802124, 0.8451126217842102, 1.2774831056594849, 0.3402882516384125, 0.3628788888454437, 2.0520734786987305, 0.8189202547073364, -1.1863313913345337, -1.9641571044921875, 0.48795732855796814, 1.085415244102478, -0.6443340182304382, -0.2550743818283081, -1.7899631261825562, -1.4778610467910767, -0.599734365940094, 0.4161206781864166, 0.04219089820981026, -1.4081268310546875, -0.124866783618927, -0.8820369839668274, 0.3569140136241913, -1.5092871189117432, -0.19679751992225647, 2.321418285369873, -1.6361836194992065, -0.19900797307491302, -0.9409142732620239, -1.3847935199737549, 2.427957773208618, -0.2928242087364197, -1.0797849893569946, -0.2691619396209717, 1.220171570777893, 1.287548303604126, 1.509937047958374, -1.0092889070510864, -0.1791263222694397, 0.5674762725830078, 2.0761566162109375, -0.4698503315448761, -1.6270867586135864, -2.1292431354522705, -0.19231460988521576, -0.23454764485359192, -0.6347657442092896, 0.03413529694080353, 0.24182547628879547, 0.09218760579824448, 0.1665833443403244, 0.5038490891456604, -1.2932254076004028, -0.8451567888259888, -0.3625837564468384, -0.4208063781261444, -0.8378294110298157, -0.30975502729415894, -0.14759542047977448, -0.3182341754436493, 0.02921193093061447, 2.5093283653259277, -0.8391176462173462, 1.4642713069915771, 1.171842098236084, -1.5063371658325195, -0.3261086344718933, 0.044165223836898804, 1.8838547468185425, 1.3376604318618774, 1.3806010484695435, 1.1925413608551025, 0.22084198892116547, -1.3843841552734375, -1.1115201711654663, -0.9277581572532654, -0.41418883204460144, 0.8578044176101685, 0.24302463233470917, -0.9163643717765808, 0.03269452974200249, 0.6277370452880859, 0.1641223281621933, 0.34353986382484436, 0.3554048240184784, -0.33859142661094666, -0.4442306160926819, -0.7330716252326965, 0.21557921171188354, 0.5512478351593018, 1.7569862604141235, -1.1505813598632812, 0.24572254717350006, -0.34057244658470154, -0.08111110329627991, -1.2242234945297241, 0.4322871267795563, -0.1443226933479309, -1.7088005542755127, -0.13844196498394012, -0.38550499081611633, 1.0354703664779663, 0.7456487417221069, 1.720258116722107, -0.01885523460805416, -0.34857386350631714, -1.4791439771652222, -0.6139125227928162, 0.907231867313385, -0.05825228616595268], [-0.14657585322856903, 1.0853736400604248, 1.740372657775879, -0.8219195604324341, 0.9483528733253479, 2.560453414916992, -0.6200816035270691, 1.1223526000976562, -0.570135235786438, 1.0791465044021606, 1.8377299308776855, 0.8952584266662598, 0.7835422158241272, -0.9017463326454163, 0.49934422969818115, 0.11503180116415024, -0.7400987148284912, 1.1791620254516602, 1.18584144115448, -1.091480016708374, -0.6967650055885315, -0.49972549080848694, 1.6881344318389893, -0.8357570171356201, 0.3048386871814728, 0.32750698924064636, 0.01921425200998783, 0.1982017457485199, -0.8116627931594849, -0.6186830997467041, -0.7904495596885681, -1.0170129537582397, 1.4461092948913574, 0.6478881239891052, 0.541530191898346, 0.4187523424625397, -0.2039278745651245, 1.8134727478027344, -0.3350788950920105, 0.9993944764137268, -0.3959903120994568, 1.0233709812164307, -0.5411818623542786, -0.4565166234970093, 1.0387042760849, 0.5698856115341187, 0.7618258595466614, 1.5474671125411987, 0.10314352065324783, -0.42381197214126587, 0.10263726860284805, 0.999315083026886, 0.16623161733150482, -0.7910593748092651, -0.03650370612740517, 0.09205107390880585, -0.07686436921358109, 0.040860865265131, -0.2249218374490738, 0.4084099531173706, 0.1277204155921936, 0.24917849898338318, 0.5288679599761963, -0.6321818828582764, -0.5974975824356079, -1.0718294382095337, -0.9349544644355774, 0.13923709094524384, -1.3709149360656738, -0.5179818272590637, 0.4067591726779938, 1.6066714525222778, 0.4899900555610657, -0.6457725763320923, 1.3291685581207275, -0.30785053968429565, -0.9155336022377014, 0.985560953617096, -0.5413616895675659, -0.02397254854440689, 0.8626099228858948, 0.908663809299469, -0.7990179657936096, -0.046184953302145004, 0.1497430056333542, -0.3392617106437683, 1.438134789466858, -1.1818673610687256, -1.0644283294677734, 0.3304813504219055, 1.6946262121200562, -0.5176008343696594, -1.1719340085983276, 1.11465585231781, -0.05443737283349037, -1.364923119544983, 1.4470494985580444, 0.5310398936271667, 0.37122905254364014, 1.137391448020935, 1.2976537942886353, -0.41608116030693054, -0.7145207524299622, -2.208231210708618, -2.05063533782959, 0.21745459735393524, 1.0674923658370972, -1.4641462564468384, -0.2875266671180725, -2.2480742931365967, 0.26368147134780884, 1.108124852180481, -2.0224602222442627, 0.1326245218515396, 1.9034816026687622, -0.11341710388660431, -0.6381062865257263, -0.1519278734922409, -0.34681791067123413, -1.1013365983963013, 1.9972212314605713, -0.8434483408927917, -1.7862423658370972, -0.8311233520507812, -0.8324559926986694, -0.8632334470748901, 0.31192371249198914, 1.2381763458251953], [-0.6105116605758667, -0.8620915412902832, -1.2595512866973877, 1.312109351158142, -0.1130482479929924, 1.573569655418396, 0.2211473435163498, -2.3792953491210938, 0.18612702190876007, 0.7465012073516846, 0.9060142636299133, -0.3453083634376526, 1.545989990234375, -0.15320655703544617, 1.3309273719787598, 0.41689008474349976, 0.28338199853897095, -0.09460636228322983, -0.3109610676765442, 0.06724962592124939, 0.22956453263759613, -1.5845454931259155, 1.2157758474349976, -1.072163701057434, 0.6633286476135254, 0.19300208985805511, 0.8030023574829102, 0.929390013217926, 2.2292604446411133, -1.8057352304458618, -0.729795515537262, -1.3777413368225098, 0.8349184989929199, 1.7850898504257202, 0.42573320865631104, -0.15369975566864014, 0.6715417504310608, 1.1711997985839844, -0.2090497463941574, 1.1829416751861572, 1.2165758609771729, -0.6566201448440552, -1.873265266418457, 0.7776688933372498, 0.8177764415740967, -0.14461825788021088, -0.7759985327720642, 1.0278232097625732, 0.6720763444900513, -0.9816784262657166, -0.7255052924156189, -0.04405815154314041, -1.0220601558685303, -0.5096112489700317, -0.3920639753341675, -0.5564721822738647, 0.1749047189950943, -0.5654404759407043, 1.6500439643859863, 1.3036940097808838, -0.14449052512645721, -0.13826356828212738, 0.36524948477745056, 1.9979093074798584, -0.30798041820526123, 0.14338858425617218, -0.38918545842170715, -0.13870041072368622, 0.1180194690823555, 0.9824406504631042, -0.7365705370903015, 0.3171015679836273, -0.4445088803768158, 0.15591315925121307, 0.7002214193344116, -2.3141870498657227, -0.1994079053401947, -0.7124063372612, -0.782303512096405, 1.4099559783935547, 0.028297748416662216, -0.8647112250328064, -2.8796005249023438, 0.2502494156360626, 0.17605112493038177, -0.47129419445991516, 0.0560440719127655, 2.2344202995300293, 0.7079082727432251, 0.8586731553077698, -1.1822599172592163, 0.16821043193340302, -1.7218639850616455, 0.9134244918823242, -0.14821214973926544, 0.5494087338447571, 0.8278576731681824, 0.5073608756065369, 1.3113032579421997, -0.1484852135181427, -0.3894851505756378, -0.18333102762699127, -0.47297701239585876, -0.018861684948205948, 0.22104546427726746, 2.8519127368927, -0.5233071446418762, -0.23847781121730804, -0.7839046120643616, 0.5856556296348572, -0.5311769247055054, 0.5291959643363953, 1.193895697593689, 0.154938206076622, -0.16361038386821747, 0.3449690341949463, 1.7898937463760376, -0.437252014875412, -0.10631293058395386, 0.6081284284591675, 0.2161875069141388, -0.8780564069747925, -1.735867977142334, -0.018683597445487976, 1.7794607877731323, 0.15218017995357513, 1.174163579940796, 0.6388445496559143], [-1.2015354633331299, -0.03438315540552139, -0.12618231773376465, 0.14590318500995636, 1.9913578033447266, 1.2975417375564575, 1.5520401000976562, 0.26269009709358215, -1.4465583562850952, 1.164676308631897, 0.642573356628418, -0.30076420307159424, 0.9034967422485352, -1.0176489353179932, -2.3629400730133057, -0.17100001871585846, -0.2160768061876297, -1.2119035720825195, -0.4152323007583618, -0.20137843489646912, 0.9453818798065186, 1.9470866918563843, 1.6135457754135132, 0.07041548937559128, 1.4575673341751099, 1.0601361989974976, 1.2081005573272705, -0.6118943095207214, -0.1537453830242157, -1.4086426496505737, 2.6101956367492676, 1.268723726272583, -1.0081946849822998, -0.1879604458808899, 1.0977084636688232, -0.5566875338554382, -0.7881967425346375, -0.16489310562610626, -0.8895082473754883, 0.026746509596705437, 0.6564587354660034, -1.3710479736328125, -0.4448983371257782, -0.4023312032222748, 2.506304979324341, -0.6617717742919922, -0.7325560450553894, -0.35856059193611145, -0.43526512384414673, -0.3901129364967346, 0.6733332872390747, -0.5068883299827576, -1.7663297653198242, -2.6262800693511963, -2.5707168579101562, -0.3437550365924835, -0.06820844858884811, 1.9938738346099854, -0.37536484003067017, -0.6617254018783569, 0.8132701516151428, 0.0728430226445198, 0.3625028133392334, 0.17710985243320465, 2.753669023513794, 0.353441059589386, -0.17880569398403168, -0.21255870163440704, 1.3429113626480103, 0.8919255137443542, 0.09265708178281784, 1.1898752450942993, 2.0513601303100586, -0.21449798345565796, 0.6308203935623169, -1.4321304559707642, 0.21843111515045166, -2.671506404876709, -0.5746821165084839, -0.5409560203552246, 1.0810332298278809, -0.43739643692970276, 1.082793116569519, -0.7423195838928223, 2.289677381515503, -0.2818671464920044, -0.16686184704303741, -0.328529417514801, -0.29529887437820435, -0.6542608141899109, -1.5800179243087769, 0.3582335114479065, -0.5034029483795166, 1.3937056064605713, 1.1346815824508667, -0.5317507982254028, 0.5782418847084045, 0.649159848690033, -1.1988621950149536, 0.671700656414032, 0.18561981618404388, -0.6830955147743225, 0.2875634729862213, -1.584854245185852, -0.008873673155903816, 0.08646636456251144, -0.03862059861421585, 0.18241417407989502, -0.13896490633487701, 0.6690903902053833, -0.1352604776620865, 0.358468621969223, -1.5657310485839844, -0.2583189308643341, 0.4027673304080963, -1.0376821756362915, 0.36602774262428284, -0.4199921190738678, 0.7999935746192932, -1.0557349920272827, -0.6026541590690613, -0.7374103665351868, -0.2897087633609772, -0.03383668139576912, 0.5373998284339905, 0.9069294929504395, 2.6634252071380615, -0.6406853199005127], [-0.48317477107048035, 1.2063976526260376, -1.1370093822479248, 0.4132654368877411, -1.842171549797058, 0.8831339478492737, -0.14794841408729553, -0.8614898324012756, -0.2879806160926819, -0.5168986320495605, -0.11027221381664276, -0.3350420892238617, 1.277782917022705, 0.25084710121154785, 0.24655215442180634, 1.0341084003448486, -0.1571403294801712, 1.6433273553848267, -0.315341055393219, 0.10253476351499557, 1.1304998397827148, -0.30671757459640503, 1.2751054763793945, 0.09048961848020554, -1.1354162693023682, 0.7571914196014404, -0.8001133799552917, 0.10829866677522659, -0.07175172120332718, -1.265440821647644, -0.7454913854598999, -0.8791895508766174, 2.1840178966522217, 0.6850072741508484, -0.2854040265083313, -0.47696247696876526, -0.8380265235900879, 0.6326632499694824, 0.5060604214668274, 1.0651603937149048, -0.08133972436189651, 2.107036590576172, 1.4584623575210571, 0.3483125865459442, -1.6130255460739136, 1.136683464050293, 0.6699661612510681, 0.17628253996372223, 0.07037326693534851, -0.877149224281311, -0.19454093277454376, 0.22978688776493073, 0.7521725296974182, -2.3236875534057617, 0.4942063093185425, 0.1099911630153656, 2.2862935066223145, -0.9745404720306396, 0.9400412440299988, 0.4583997428417206, -0.22371326386928558, 0.41381534934043884, 1.4315285682678223, 1.031562089920044, -1.7065454721450806, 0.45232322812080383, 1.939608097076416, 1.737123727798462, 1.177354335784912, 0.6711588501930237, 1.1245626211166382, -0.361390620470047, 0.7887431383132935, 0.6498393416404724, 3.177863597869873, 0.31136587262153625, 0.8435571789741516, -2.483541488647461, -0.5141120553016663, -0.7275004386901855, 0.08361982554197311, 2.496994972229004, -0.6357209086418152, -1.214368462562561, 0.40051698684692383, 1.0635788440704346, -0.6536775827407837, 1.6290745735168457, -0.4435541033744812, -0.591393232345581, 0.5134879350662231, -0.6172550916671753, 0.07354214787483215, -0.7177213430404663, -0.5904306173324585, 0.18499965965747833, 0.3322506248950958, 0.13451796770095825, 0.7538241147994995, -0.26941442489624023, -1.3657877445220947, -0.24870453774929047, 0.4542613923549652, 3.2336905002593994, -0.48457613587379456, 0.4641430974006653, 0.3533523678779602, -0.5391764044761658, -0.24610239267349243, 1.1760609149932861, -0.000885139568708837, 0.31995299458503723, 0.1704975664615631, 0.7224692106246948, -0.7488270401954651, -0.11549568176269531, 0.5769011974334717, 0.016396183520555496, 1.8479710817337036, -0.25508928298950195, 0.5594375729560852, 0.8084591627120972, -0.046137142926454544, 0.4857467710971832, 1.1141585111618042, -0.7602895498275757, 0.5423335433006287, -0.1345590204000473], [1.4785315990447998, -1.9947744607925415, 1.035404086112976, -0.023930154740810394, 0.539665937423706, 1.3415476083755493, 0.49977296590805054, -0.15866854786872864, -0.7304139733314514, 1.4732030630111694, 1.2184011936187744, 0.304844468832016, -0.45363447070121765, -0.06362591683864594, -0.26914653182029724, 1.0819175243377686, -0.06996507942676544, -0.8216292262077332, 1.2755701541900635, -0.5411286950111389, 0.4188859462738037, 1.11823308467865, 1.3821070194244385, -0.14772717654705048, 0.04395073279738426, 0.13197025656700134, -1.1744344234466553, -0.08137798309326172, -0.15896055102348328, -1.2542957067489624, -0.765541136264801, 0.13333870470523834, 1.3468804359436035, 0.44364669919013977, -1.56788170337677, 0.9261549711227417, -1.4333744049072266, 0.7093204855918884, -1.88382089138031, 2.0862507820129395, 0.32756903767585754, -0.5289273262023926, -0.8217220902442932, 0.6182669401168823, 0.6831401586532593, 0.8008584380149841, 0.11204732209444046, 2.0334250926971436, -0.11100360006093979, -0.019443221390247345, -0.8919054269790649, 0.5641994476318359, -0.9533023238182068, 0.7680732607841492, -1.1058399677276611, 0.24669794738292694, -0.010929681360721588, -1.0779047012329102, -0.1370951235294342, 0.12194524705410004, 1.5236181020736694, 0.8151165246963501, 0.4064783751964569, 1.4937447309494019, 0.812578558921814, 0.6155499219894409, -0.4978022873401642, -0.023732183501124382, 1.7518789768218994, 0.6793406009674072, -0.406246155500412, -1.0061091184616089, -0.35263171792030334, 1.769274353981018, 0.3864819407463074, -0.30203837156295776, 0.6179229021072388, -0.4693567752838135, 1.0016257762908936, 0.7815403938293457, 1.8248902559280396, 2.2539889812469482, 1.4079498052597046, -0.3641220033168793, -0.09529147297143936, -0.7617889046669006, -0.17041422426700592, 1.001550316810608, 1.2842674255371094, -1.3306794166564941, -0.3311079740524292, -1.6471790075302124, -0.5434482097625732, 0.2809085547924042, 0.09068857133388519, -0.52098548412323, -0.10174672305583954, 1.7493867874145508, 0.26254889369010925, -2.0000433921813965, -0.6560883522033691, -0.840823769569397, 0.06116080656647682, 1.0287280082702637, -1.539453148841858, -0.09655004739761353, 0.4464735686779022, -1.0523418188095093, 1.4909290075302124, 1.1351432800292969, -1.1044971942901611, -0.15939490497112274, 0.6213821172714233, -1.1025680303573608, 1.2234169244766235, 1.153638243675232, 1.4432634115219116, 0.38601160049438477, 1.195439100265503, -0.4632783532142639, -0.1346472054719925, 0.6916937232017517, 1.0275276899337769, 0.8772457242012024, 0.6265745162963867, 2.205428123474121, -0.6993874907493591, 1.0456466674804688], [-0.302890807390213, -0.7346047759056091, 1.849030613899231, -1.0543959140777588, 1.6144640445709229, -0.22787164151668549, -2.481997489929199, 0.8500804305076599, 0.16164082288742065, -1.351027011871338, -1.9876031875610352, -0.9350428581237793, -2.3304781913757324, -0.48799753189086914, -0.4073021113872528, -0.6427815556526184, 0.930496335029602, 1.338823676109314, 1.0713882446289062, 2.3176767826080322, -0.25823715329170227, -1.5922621488571167, -3.021684408187866, -0.017203018069267273, -0.23254309594631195, 0.328541100025177, 0.2425384670495987, -1.3123537302017212, 1.291748046875, 1.70571768283844, 1.5299493074417114, -0.4842798411846161, -2.3735525608062744, -0.14221805334091187, -0.4239661693572998, -0.658913791179657, -0.5502520203590393, 0.8257439136505127, -0.08309320360422134, -0.15531137585639954, -1.852668285369873, -0.3382142186164856, -1.360298991203308, 0.642045259475708, -0.8339729309082031, 0.12027998268604279, -0.6817634105682373, 0.4606766700744629, -0.9282940626144409, -0.6696243286132812, 0.8432275056838989, -1.0576261281967163, -0.9250885248184204, -0.9081042408943176, 0.6708317399024963, -0.7224358916282654, -0.5613935589790344, -0.9631156921386719, 0.7712850570678711, 0.0020808337721973658, 2.3859593868255615, -0.7927306890487671, -0.7108796834945679, -0.8892886638641357, 0.671221137046814, -0.2672581672668457, 0.4921607971191406, 1.7559601068496704, -1.828999400138855, -1.8430736064910889, -1.8141427040100098, 1.3768495321273804, 0.3231251537799835, -0.8027843832969666, -0.002951415255665779, -0.5703358054161072, -0.6285774111747742, -1.2245090007781982, 0.012990999966859818, -0.6095225214958191, -0.5731873512268066, -0.7601644396781921, -0.1406405121088028, 0.18033252656459808, 0.538331151008606, -0.728469729423523, 0.3581506609916687, -0.2774501442909241, 0.03288460895419121, 1.5171091556549072, 0.6548228859901428, -0.5718388557434082, 0.5542213916778564, -1.6877241134643555, 0.1435157060623169, -0.5061380863189697, 0.25293830037117004, 2.352586269378662, 0.36050963401794434, -0.5504634380340576, 0.10267201066017151, 0.9246860146522522, 0.6694542765617371, 1.4357694387435913, 1.6100707054138184, -0.6591658592224121, 1.5322535037994385, -0.9263378381729126, -0.7332174181938171, -0.34135568141937256, -1.9595164060592651, 0.7013403177261353, 0.25039416551589966, 0.9757835865020752, 1.029598593711853, 0.48387834429740906, -0.4969770610332489, -0.5436409711837769, -1.1152197122573853, 1.3798202276229858, 1.8051961660385132, -1.0740703344345093, -1.7007503509521484, -0.14665314555168152, 0.8793508410453796, -0.9247903227806091, 0.042922258377075195, 1.1012264490127563], [1.1666994094848633, -0.682389497756958, 0.30501148104667664, 0.011464783921837807, -0.07337117940187454, 0.21485547721385956, 0.4881264865398407, -0.08749239891767502, 2.8729074001312256, -0.28537940979003906, 0.6597089171409607, 1.2238982915878296, -0.578691840171814, -0.24583984911441803, 0.15844261646270752, -0.3275676667690277, 0.19831062853336334, -1.0955814123153687, 0.2283661663532257, 0.8247671127319336, -1.2387593984603882, 1.4497673511505127, 0.14042428135871887, -0.9054052829742432, -0.14874118566513062, -1.2554572820663452, -1.2635128498077393, 0.43777358531951904, -0.9185992479324341, 0.28261876106262207, 0.28226813673973083, -0.11478057503700256, 0.6367691159248352, -1.0448416471481323, 1.0870407819747925, 1.9610440731048584, 1.3631923198699951, -0.16500738263130188, -1.1190170049667358, -0.7840004563331604, -0.10884305834770203, 1.3355320692062378, 0.13766270875930786, 2.396052360534668, -1.0370070934295654, 0.8640665411949158, 0.5972021818161011, 0.4115371108055115, -1.688256025314331, -1.4159504175186157, 0.0005158746498636901, 1.7139368057250977, -0.6702373623847961, -1.197300910949707, -0.4355831742286682, -0.2882407307624817, 0.30861374735832214, -0.8523789644241333, 0.3299390375614166, -1.17103910446167, -1.0691863298416138, 0.0014903091359883547, -2.563063144683838, 0.5240805149078369, -0.7060211300849915, 1.0773779153823853, 1.7341479063034058, -0.3950415253639221, 0.3071550130844116, 0.29269587993621826, -0.050137925893068314, 0.10442805290222168, 0.42036065459251404, -1.6792048215866089, 0.9381673336029053, 0.816633939743042, -0.5671420097351074, -2.4711437225341797, 1.261414885520935, 0.49435004591941833, -0.6595271825790405, 2.1010398864746094, 0.7766369581222534, -0.0909646674990654, 0.884450376033783, -0.7342808842658997, 0.5241814255714417, -0.29432758688926697, 0.5625408887863159, 0.4067055881023407, -0.787007749080658, -0.006353143602609634, -0.5425040125846863, 0.45567968487739563, -1.199016809463501, 0.24907974898815155, 0.13024991750717163, -1.0672070980072021, 0.1696908324956894, 0.4610489308834076, 0.5551972389221191, -0.5084289908409119, 1.297930121421814, 0.8091283440589905, 0.8265956044197083, -0.1897398829460144, -0.58564692735672, 1.8019205331802368, -1.250779628753662, -0.7073912620544434, -0.32025444507598877, 0.39058127999305725, -1.6772841215133667, -0.47069379687309265, 0.9655018448829651, 1.156122088432312, 0.3288012444972992, -0.5091432332992554, 0.7824015617370605, 1.573166012763977, -0.43532583117485046, 0.6022011041641235, 0.7773742079734802, -0.9240368604660034, 0.06963372975587845, 0.9441589713096619, -0.8023662567138672, -1.1123889684677124], [0.04204215109348297, 2.3455512523651123, 0.8970682621002197, -0.025565410032868385, 0.09443916380405426, 0.08246073871850967, -0.11052829027175903, 0.6528158187866211, 0.08345264941453934, 1.362221360206604, -0.3874123692512512, 2.299452066421509, -1.3050600290298462, 0.9150518774986267, 0.17718109488487244, -1.1092536449432373, -2.1824440956115723, -0.1313023418188095, -0.44130459427833557, 0.26934367418289185, -1.2635650634765625, -0.9911187291145325, 0.06022130325436592, -0.1329270750284195, 0.06512501835823059, 1.298175573348999, -0.563105046749115, -0.1369355022907257, -1.5639995336532593, -0.5628741383552551, -0.6242926120758057, -0.08615333586931229, 0.5829777121543884, 0.7926374077796936, -1.6595735549926758, -2.431251287460327, -0.6563833951950073, -1.3346954584121704, -0.7719864249229431, 0.07062830775976181, 0.07221560180187225, -1.667518138885498, 0.5088359117507935, -0.488739937543869, 0.8943397402763367, 0.062334951013326645, 0.8310433030128479, 0.36226844787597656, 0.2524877190589905, -1.0435125827789307, 1.075627326965332, -1.175489068031311, 2.6279122829437256, 0.452619731426239, -0.8738347887992859, 0.5983237624168396, -1.8191510438919067, -0.5963839292526245, 0.7685343027114868, 1.0377533435821533, -1.1639772653579712, 0.4850817620754242, -0.660116970539093, 0.6980565190315247, 0.0757298395037651, 0.3497159481048584, 2.047727346420288, -0.7550932168960571, 0.15307794511318207, 0.4960636794567108, -1.0184211730957031, 0.7619292140007019, 0.922469973564148, -0.42499494552612305, -0.4881596863269806, 0.422505259513855, 0.35473278164863586, -0.38317441940307617, -0.0015491573140025139, 0.7753188610076904, 0.5333759188652039, -1.0468964576721191, -0.702425479888916, -0.27604612708091736, -0.040470968931913376, 1.0301185846328735, -1.7667187452316284, -1.864402174949646, -1.2683539390563965, -0.5149946212768555, -0.42611685395240784, -0.41881048679351807, -0.06582188606262207, -0.19380682706832886, -0.3621375858783722, 0.4258463382720947, -0.6613137722015381, -0.1481175422668457, -0.5091860294342041, -0.42624613642692566, 1.000911831855774, -0.07619918137788773, 2.7477917671203613, -0.09853452444076538, 0.5432152152061462, -0.7339640259742737, -0.40958642959594727, -0.21238099038600922, 1.1342064142227173, -1.480826497077942, 1.0874879360198975, -0.2705768942832947, 0.2411080300807953, -0.7390091419219971, -1.6228729486465454, -0.10045433789491653, -0.9084652662277222, -0.2541818618774414, -0.8505715727806091, -2.615732192993164, 0.7777314186096191, 0.48709163069725037, -0.2848047912120819, 0.10148277878761292, -1.139548420906067, -0.734738290309906, 0.11271556466817856, 0.6292660236358643], [1.0150482654571533, 2.3685121536254883, 0.26090121269226074, 1.4404839277267456, -0.9254457354545593, -1.2598832845687866, -3.0715179443359375, -0.175298810005188, 0.9887726902961731, 2.3415350914001465, -0.6611824035644531, -1.2333871126174927, -0.37917304039001465, -1.2366584539413452, -0.2491602897644043, 0.9345605969429016, -1.1316484212875366, -0.9411136507987976, 1.5915112495422363, 1.1696535348892212, 0.5207183361053467, 0.7578502297401428, -1.3810456991195679, -0.49197468161582947, -0.15891888737678528, 0.8945956230163574, 0.5298519134521484, 0.4360228478908539, 2.773580312728882, -0.4430205523967743, 0.19641254842281342, -0.6154664754867554, -0.030713148415088654, 0.6742561459541321, 0.5541030168533325, -0.5344188213348389, -0.5746934413909912, 0.2878466844558716, -2.0064804553985596, 2.344871997833252, -0.7345776557922363, -0.0853591337800026, -0.3639392554759979, -0.6729533672332764, -0.018446318805217743, 0.1868305206298828, 0.8304049968719482, -0.13297641277313232, 1.0872087478637695, 0.2577337920665741, 0.16655971109867096, -0.25056976079940796, -1.2152621746063232, -0.8479613065719604, 0.880855917930603, 1.3289674520492554, 0.7726956009864807, 0.8758741617202759, 1.0116071701049805, -0.8813362717628479, -0.4928302466869354, 0.8529753088951111, -0.37724393606185913, -0.6479282975196838, -0.48847460746765137, 0.6416210532188416, -0.4029584527015686, -1.0371266603469849, -1.3881630897521973, 0.34669941663742065, -0.006869150325655937, -0.9967716932296753, 0.7828903794288635, -0.3906222879886627, -0.25929391384124756, -0.9640308618545532, -0.5886707901954651, 0.0035415329039096832, -0.7302257418632507, 0.2052316963672638, 0.4596216678619385, -0.46309372782707214, 2.560413122177124, -0.08386976271867752, -1.902870774269104, -0.3764428198337555, -1.1737602949142456, 2.9970338344573975, 0.7254726886749268, 0.30845093727111816, 1.1956582069396973, 0.19386447966098785, -0.13563379645347595, 0.011824913322925568, 2.179985523223877, -0.8895683288574219, 0.1367684304714203, -0.8811843991279602, 0.4964783191680908, -0.6569503545761108, 0.9358485341072083, -1.0807263851165771, 1.2769607305526733, -0.9227799773216248, -0.6405137777328491, 1.2206921577453613, 0.3822135627269745, 0.3141067326068878, 0.8851595520973206, 1.8779399394989014, -0.8632287979125977, -0.9007693529129028, -0.7849317789077759, -0.3973856270313263, -0.6388971209526062, -1.001334547996521, -3.22810697555542, -0.8449379801750183, -0.7366739511489868, 0.7477745413780212, -0.07409127056598663, -1.6778428554534912, -0.38059213757514954, 0.7413339614868164, -0.870283305644989, -0.6374297142028809, -0.6414198279380798, -0.4527004361152649], [0.018370093777775764, 0.981347918510437, 1.1918978691101074, 0.3411658704280853, 0.5468876957893372, 1.212722659111023, -0.576922595500946, 0.09786520898342133, -0.04503894969820976, 0.5302399396896362, -2.248276948928833, -0.7314016819000244, 1.2652403116226196, -0.6185941696166992, -0.6392077803611755, -0.23545047640800476, -0.3352416157722473, -0.7570230960845947, 1.7761179208755493, -0.16121654212474823, -2.9846560955047607, 0.6908182501792908, -1.1893556118011475, -0.9710906744003296, 0.6905468106269836, -0.2741866111755371, -0.30826088786125183, -0.7773504257202148, 1.4349799156188965, 0.7104278802871704, -0.5734219551086426, -0.24672174453735352, 0.5830478072166443, 0.08383975923061371, 1.0099502801895142, 1.5961681604385376, 0.07977856695652008, 1.016357421875, -2.4258549213409424, -1.7113701105117798, 1.2826257944107056, -0.6303467750549316, 0.7002348303794861, 2.254878044128418, 1.7302501201629639, 0.8602017164230347, 0.9108095765113831, -0.6747487783432007, 0.3979474902153015, 0.11001951992511749, 0.7481228709220886, 1.1539506912231445, 1.1427665948867798, -0.18005456030368805, 0.2927878499031067, 0.8485921025276184, -0.14189459383487701, 0.22369027137756348, -0.5539240837097168, 0.4912218153476715, 0.3618318438529968, 0.5334158539772034, -0.34987136721611023, -1.0077588558197021, 0.40139755606651306, 0.257750004529953, 0.74993497133255, 0.08153661340475082, 1.5755455493927002, 0.7393434643745422, -1.392289638519287, -0.13034558296203613, -0.28909438848495483, 0.9178543090820312, -0.011135688982903957, -1.3253324031829834, 1.2798941135406494, 0.5250806212425232, -1.8129717111587524, -0.5524746775627136, -0.9178186655044556, 0.7951949834823608, -0.5689409971237183, 0.6525307893753052, 0.2964743673801422, -2.101473808288574, -2.2007431983947754, 0.4372819662094116, 0.2687051594257355, 0.9973447322845459, -1.1382900476455688, 1.801936388015747, -1.0202805995941162, 0.4254268705844879, -0.5121206045150757, -0.1531641185283661, -0.2163529098033905, -0.7072530388832092, 0.7359932065010071, 0.7816153168678284, 1.2937812805175781, 0.4683188199996948, 0.6333878040313721, -0.5231876969337463, 0.6209057569503784, 0.9947025775909424, -0.32257434725761414, 1.2756901979446411, 0.2903549075126648, -0.7024857401847839, -0.0945628434419632, -1.2683552503585815, 0.7531015872955322, 0.08904925733804703, -0.014360075816512108, 0.2231101244688034, -0.8284494280815125, 1.0869866609573364, -0.747132420539856, 0.40858015418052673, 0.3219620883464813, -0.1276070773601532, 0.14025025069713593, -0.24261920154094696, -0.6395967602729797, 0.22784318029880524, 1.4625555276870728, -1.032355546951294], [0.6558860540390015, 1.174971342086792, 0.46655336022377014, -0.8438834547996521, -0.07037888467311859, -2.319164514541626, 1.0671558380126953, -0.37348097562789917, -0.5687845349311829, 0.8930564522743225, 1.32789945602417, -1.020944356918335, 0.5887264013290405, -0.5035668611526489, 0.5242671966552734, -0.6370870471000671, -0.6230701208114624, 0.8876034021377563, -0.32451990246772766, -1.1861965656280518, -0.9615036845207214, -0.6909343600273132, -0.7132254838943481, -0.6372560262680054, -0.5158503651618958, 0.3752545416355133, -1.9275224208831787, -0.14140449464321136, -1.027404546737671, 0.4161900281906128, 0.1548735648393631, 0.4845130741596222, 0.1886209100484848, -0.6543164253234863, 0.5495124459266663, -0.1428484469652176, -0.7991943955421448, 0.918725848197937, -1.091775894165039, 0.9472223520278931, -1.6607249975204468, -0.075648732483387, -0.60762619972229, -1.5554816722869873, -0.45823681354522705, 0.5162692070007324, -0.396028995513916, 1.291154384613037, 1.3487190008163452, 0.8649289608001709, 1.4278569221496582, -1.6161727905273438, -1.2092491388320923, -0.2187834084033966, 0.9805787205696106, -0.36321884393692017, 0.4069141447544098, 1.7117488384246826, -1.0540471076965332, 2.7972469329833984, -0.270043283700943, -1.0085924863815308, 0.3651098906993866, -0.9970203042030334, -2.5185775756835938, -2.6271495819091797, -0.2819182872772217, 0.19743677973747253, -2.8844361305236816, 0.6930203437805176, -0.7969970703125, -0.6910919547080994, 0.5311597585678101, -0.5830419063568115, -1.1843514442443848, -1.474183440208435, -0.16576805710792542, 0.781015932559967, 1.2181447744369507, -0.48340150713920593, 1.8734387159347534, 2.3151683807373047, -2.080519914627075, -0.5146113038063049, -0.43068450689315796, 0.2974366247653961, 0.0748402327299118, 0.03568238392472267, -0.0494028739631176, 0.047550003975629807, -0.6009352207183838, 0.14154835045337677, -0.1255568563938141, 0.5473344922065735, -1.5365928411483765, 0.44883543252944946, -0.7561939358711243, 0.882566511631012, -0.5943854451179504, 1.3333241939544678, 0.21203091740608215, -1.4096381664276123, -0.23601599037647247, -1.8378576040267944, 2.312307357788086, -0.4760018289089203, -1.7932970523834229, -0.6480270028114319, -0.6333038806915283, 0.5187981724739075, 0.8419368863105774, 0.6913281679153442, -0.7071930170059204, -0.6396369338035583, -0.9913558959960938, -1.0342086553573608, -0.40212002396583557, 0.21092352271080017, -0.3709118664264679, -0.05256282165646553, -1.0147249698638916, 0.6558261513710022, -0.5906877517700195, -0.6593948006629944, -0.2809579074382782, 0.0499100387096405, -0.2345544397830963, 1.0292954444885254], [1.2146618366241455, 0.1406034380197525, -0.08255708962678909, -0.36404672265052795, 0.39278310537338257, -0.5572306513786316, -0.12074550241231918, -1.3128048181533813, -0.09207146614789963, -1.2143609523773193, -0.5817254185676575, 0.4701248109340668, 1.4408352375030518, -0.48406657576560974, 0.3969576060771942, 1.1164830923080444, 1.4964038133621216, -0.8993239402770996, -0.2088085263967514, -2.131053924560547, 2.2138893604278564, -2.2766339778900146, -0.14710400998592377, -0.4486640393733978, -0.35366132855415344, 0.943432629108429, 1.4908794164657593, 1.2519420385360718, 0.8250814080238342, -0.8093432784080505, 2.3943560123443604, 0.6445341110229492, -0.6984551548957825, -1.270682692527771, -0.10094040632247925, 3.120651960372925, 0.25573110580444336, -0.2513105869293213, -0.25168561935424805, -2.1436355113983154, 0.010299254208803177, -0.5980501174926758, 0.1318025141954422, -2.1511940956115723, 0.7497549057006836, -0.13191765546798706, -0.3669810891151428, 0.3922550678253174, -0.6090688109397888, -0.42193567752838135, 1.9141069650650024, -0.05144686624407768, 0.8293282389640808, -0.5826703310012817, -0.6959807872772217, -0.05405318737030029, 0.7916380763053894, -0.3738202750682831, -1.301410436630249, 1.4807510375976562, 1.9902530908584595, 0.12747429311275482, 0.023356765508651733, -0.50437331199646, -0.6682530045509338, -1.3872393369674683, 0.6474326252937317, 0.7047035098075867, -0.7344654202461243, -0.4441177248954773, -0.5358996391296387, -0.8387807607650757, -1.3963985443115234, -0.064698226749897, -0.4433014988899231, 0.04103961959481239, 0.8533671498298645, 0.7398542165756226, 0.015704309567809105, 0.1709500551223755, -1.2390280961990356, -0.7152327299118042, 0.5581124424934387, -0.5726686120033264, -0.20179316401481628, -0.24390986561775208, 0.29739752411842346, -0.06739561259746552, -2.144437789916992, 0.9756144881248474, 0.5682201385498047, 0.08684837073087692, 0.47430065274238586, -0.2726074159145355, -0.33938074111938477, 1.3194152116775513, 0.5351572036743164, 1.9353270530700684, 0.6633940935134888, 0.14320065081119537, -0.3333892822265625, 0.04949120804667473, -0.4509100019931793, 0.2425585836172104, 0.5975503921508789, -0.21034149825572968, 0.9352289438247681, 0.7358525395393372, 1.017276644706726, -0.7930224537849426, 1.3497989177703857, 2.6848413944244385, -1.1194138526916504, 0.660205066204071, 0.9577228426933289, -0.3735068440437317, 1.3959965705871582, -0.07680169492959976, -0.5733289122581482, 1.1564571857452393, -1.746058702468872, -0.0389213040471077, -0.3055468499660492, -0.4499782919883728, 0.41316965222358704, 0.5912085771560669, 1.3767086267471313, -0.12405235320329666], [0.49907708168029785, -0.13148289918899536, -0.5222135782241821, -0.9755010008811951, 0.9909220337867737, 1.189552903175354, 0.33314165472984314, -0.40315040946006775, 0.029547257348895073, -0.41916513442993164, 1.1777862310409546, -1.6553246974945068, 1.4615031480789185, 0.1424494832754135, 1.0033711194992065, -0.14559635519981384, -1.8320820331573486, 2.02626895904541, -0.7282776832580566, -0.5411430597305298, -0.5113601684570312, 0.7299449443817139, 0.712398886680603, -0.8783344626426697, -1.2212989330291748, -0.41575390100479126, -0.014614484272897243, -0.2465437650680542, 0.33389610052108765, -0.3185862898826599, -1.0035027265548706, 1.274870753288269, 1.1215510368347168, 0.5077656507492065, -1.941635012626648, 1.6161599159240723, -1.1685640811920166, 0.6643678545951843, 2.039663553237915, 0.16571833193302155, 0.04430393502116203, 1.6562381982803345, -0.7573091983795166, 0.3639833331108093, 0.045193858444690704, -1.09992516040802, 1.9198508262634277, 0.3607173562049866, 0.27865850925445557, -0.9733843803405762, -0.8246891498565674, -0.4321768283843994, -0.493726521730423, -0.18683640658855438, 2.031385898590088, 1.363128423690796, -0.117239810526371, -0.24799545109272003, 1.8052362203598022, 1.6665689945220947, 0.05467066913843155, 0.4184550344944, 0.29663407802581787, -0.7024971842765808, 0.8050379753112793, 0.50279700756073, -0.12965907156467438, 0.32097846269607544, 1.0735286474227905, -0.3674105703830719, 2.024596929550171, -2.0248963832855225, 0.6443231701850891, 1.3033361434936523, -2.179699420928955, -0.04624967277050018, -2.1183154582977295, -1.7398375272750854, 0.6223823428153992, -0.47428855299949646, 1.6452655792236328, -1.0737468004226685, 0.3406449854373932, 0.44080281257629395, -0.7889440655708313, -1.5988664627075195, -2.034615993499756, 0.8608916401863098, 1.9932290315628052, -0.222802996635437, -0.7890612483024597, -0.4495091438293457, 0.8937336802482605, -0.3763517141342163, 0.11786264181137085, -1.3614351749420166, 0.27775251865386963, -0.6394072771072388, -1.3034229278564453, 0.2966209948062897, 1.3407713174819946, -0.0763750672340393, 0.011056870222091675, -0.29267996549606323, -0.854263961315155, 0.22017760574817657, -0.22257712483406067, 1.493971824645996, -0.7919903993606567, -1.8463290929794312, 1.7284824848175049, 0.670827329158783, 0.38957586884498596, -1.2681264877319336, 0.5727189183235168, -1.194061279296875, -1.6531611680984497, 0.4757082462310791, -0.9410219788551331, 1.3026585578918457, -0.14118725061416626, 0.17769168317317963, 0.8913666009902954, -1.6482758522033691, 2.054534435272217, -1.7955824136734009, 0.7562653422355652, 0.13924169540405273], [0.16247646510601044, -0.3185007870197296, 0.651008665561676, 0.3041408061981201, -0.35582250356674194, 0.31360140442848206, 0.5891240239143372, -1.8698716163635254, -0.42480504512786865, -1.388580560684204, -0.14442817866802216, -1.6174795627593994, 0.3797883093357086, -0.7884772419929504, 0.2269665002822876, -1.7781208753585815, 0.0129482327029109, 1.374318242073059, 0.33363625407218933, -1.5424790382385254, -0.36838844418525696, 0.9034259915351868, -1.5585548877716064, -0.47727787494659424, 1.211747407913208, -0.8271144032478333, -1.126164436340332, 0.182374969124794, 0.4834650456905365, 1.0707464218139648, 1.3572440147399902, 0.6155372262001038, -0.5916313529014587, -2.039015769958496, -0.7102518081665039, -0.08283548057079315, 0.740825891494751, -0.2564786374568939, 0.07438068836927414, 0.8018339276313782, -1.1663175821304321, -0.5339807271957397, -0.7314466238021851, -0.5832487940788269, 0.37660014629364014, 1.6420260667800903, 1.7919093370437622, 1.4345922470092773, 0.3322972059249878, -0.6732217669487, -0.6829022169113159, 0.9983729720115662, 0.5194718837738037, -0.5869448781013489, 0.6817993521690369, 0.287070631980896, 1.0860280990600586, -1.0571962594985962, 0.28045135736465454, 0.07366891205310822, 0.6518696546554565, -0.45777973532676697, 0.14507576823234558, 0.2521374523639679, 0.36660003662109375, 0.8390006422996521, -0.5816071033477783, 2.6316072940826416, -1.9106268882751465, 0.6368409991264343, 0.22329305112361908, -0.42605361342430115, -1.1461659669876099, 2.210855484008789, 1.2510855197906494, -0.7226350903511047, -0.38259226083755493, 0.17462629079818726, 0.3068437874317169, 0.2418387532234192, 0.874828577041626, 0.2700229287147522, 1.4660301208496094, -0.3609456419944763, 0.3037993013858795, -0.5839307904243469, -0.8597473502159119, -0.4409627914428711, 0.778715193271637, 0.4282072186470032, 1.5897350311279297, 0.4534217417240143, 1.0925791263580322, -1.1046029329299927, 0.048679906874895096, 2.8166565895080566, 1.3412141799926758, -0.06252215802669525, 0.9585594534873962, 0.4289894700050354, 1.97769296169281, -0.5407934784889221, 1.7449787855148315, 0.8815553784370422, -0.5839366316795349, 1.1513758897781372, 0.06409358233213425, 2.449246883392334, -0.9867829084396362, 0.4681706726551056, 0.02323685586452484, 1.9148766994476318, 0.06527242064476013, 0.3060680329799652, 0.35177356004714966, -0.2769688665866852, -0.7464302778244019, 0.04386477172374725, -0.4918768107891083, -0.3867512047290802, -0.27460166811943054, -0.6777183413505554, 0.00035954429768025875, -1.216381549835205, -1.15938138961792, -0.837783694267273, 0.03378111124038696, -0.0002770089195109904], [-1.7982381582260132, -0.07412106543779373, -0.0739273726940155, -0.4044070243835449, 0.9866806864738464, 0.7898138165473938, 0.9922500848770142, -0.11196759343147278, -0.7137539386749268, 1.0067192316055298, -0.6683311462402344, -1.3914768695831299, 1.1010159254074097, 0.4517515003681183, 0.07729707658290863, -0.37699270248413086, 0.25605538487434387, -1.3932749032974243, -0.04259004816412926, -0.22269611060619354, -1.623633861541748, -0.48245659470558167, -0.4505494236946106, -0.5912887454032898, -1.052377700805664, -1.115356206893921, 1.544451355934143, 0.4958398938179016, -1.1649377346038818, 0.5324690937995911, 0.0658905878663063, 1.4018162488937378, 0.9331876635551453, 1.87577486038208, 0.0757436603307724, -0.4480714201927185, -0.552569568157196, 0.8531579375267029, 0.9605311155319214, -0.6057400107383728, -1.4784609079360962, -0.3598374128341675, 0.06409069895744324, -0.06030138581991196, 0.18741583824157715, 1.2794846296310425, -0.38919419050216675, -0.014323040843009949, -0.22555434703826904, -0.9702747464179993, 0.43437108397483826, 0.08015379309654236, 0.12209053337574005, -0.18607649207115173, 0.07197783142328262, 0.607115626335144, 1.2551922798156738, -1.098522424697876, 0.9233614802360535, -0.013301244005560875, -0.20549942553043365, 0.6412755250930786, 0.3669413626194, 1.1888889074325562, -0.004269338678568602, 2.154836654663086, -1.1806883811950684, -1.078033447265625, -1.7000501155853271, -0.03514023870229721, -1.5877641439437866, 1.676276683807373, 0.649555504322052, -0.4508219063282013, 2.3754193782806396, -1.4756823778152466, -1.467233657836914, 0.07767633348703384, 0.4261743128299713, -0.6230730414390564, -0.0737794041633606, 0.4292415678501129, -0.16208723187446594, 0.6819891333580017, -1.434476613998413, -1.9124157428741455, 1.2222049236297607, -0.4182019531726837, -1.195033311843872, -0.39483368396759033, 1.3891587257385254, -0.8795579671859741, 0.12604781985282898, -0.11823461204767227, 0.13390497863292694, 1.5647813081741333, 1.7697848081588745, 1.1664339303970337, 1.2942512035369873, 0.09423190355300903, 0.596678614616394, -1.9692164659500122, 1.3337244987487793, -1.5212820768356323, -1.4520107507705688, -0.3837067186832428, -0.004597553983330727, 0.3011361360549927, 1.2709286212921143, 0.15225045382976532, -0.5241258144378662, -0.4711146950721741, 0.3955595791339874, -0.7618789076805115, 1.6031339168548584, 0.4460288882255554, 0.6723383665084839, -1.6217758655548096, -1.6356720924377441, 1.070359230041504, -0.44386231899261475, 0.6767390370368958, -0.1373448520898819, -0.11135950684547424, 2.1067922115325928, -1.5745781660079956, -0.7383987307548523, 2.095205307006836], [-1.7917295694351196, 0.06483094394207001, -2.4310672283172607, 0.08904213458299637, -0.37273040413856506, -0.5037652254104614, -0.0645654946565628, -0.4101739525794983, 0.38274523615837097, -0.006320103071630001, 0.46159452199935913, 0.8763110637664795, 0.7371581196784973, 0.5348299145698547, 0.9932968020439148, -1.066046118736267, -0.7355870008468628, -1.3393229246139526, 0.7069936394691467, -0.6795548796653748, 0.9408150911331177, 0.043866366147994995, -0.7507796883583069, -0.21586468815803528, 0.4320356547832489, -1.3833208084106445, -0.15915250778198242, 2.2989869117736816, -0.38758400082588196, -1.693739414215088, -1.7341854572296143, -1.217643141746521, -0.26485633850097656, 0.3709617257118225, -1.2395005226135254, 1.1148061752319336, 0.9303411245346069, 1.4536025524139404, -1.2740188837051392, -0.11610181629657745, -0.7803534865379333, -0.4033587872982025, 1.6964120864868164, 0.5003424286842346, -2.3869516849517822, 1.7352159023284912, -1.1688066720962524, -0.00013454009604174644, -0.2782997190952301, 0.2256341278553009, -0.899937629699707, -0.9424335956573486, -1.6172178983688354, -1.2605550289154053, -0.12361757457256317, -0.870994508266449, 1.644448161125183, -0.5509184002876282, -0.844875693321228, 1.7949858903884888, -0.6028797626495361, -0.8972577452659607, 0.5603628158569336, -1.3278712034225464, -0.13033345341682434, 0.7796822190284729, 0.7237809300422668, -0.026046769693493843, 1.362163782119751, 0.6209612488746643, 0.1610969752073288, -0.4039102792739868, -1.2854844331741333, -0.6698138117790222, 0.055918388068675995, 0.9891256093978882, 0.6955485939979553, 1.219967246055603, -1.588173508644104, -0.7714108824729919, -0.4086419343948364, -2.1944668292999268, -0.07320721447467804, 1.202987551689148, 1.5153394937515259, -0.14735578000545502, 0.09864180535078049, 0.6581971645355225, -0.10833572596311569, -0.46948227286338806, 1.6859363317489624, -0.6561222076416016, -0.5291541814804077, -1.0495879650115967, -0.46541351079940796, 0.7702771425247192, -0.12791386246681213, 0.8916618824005127, -0.1196703240275383, 0.2926722466945648, 0.3107839822769165, -0.5942073464393616, 0.7208015322685242, 0.6902475953102112, -0.815623939037323, 0.649850070476532, 0.5108404755592346, -0.6873063445091248, 1.3854455947875977, -0.11509236693382263, -0.7296644449234009, 0.09899970889091492, -0.6843710541725159, -0.13832339644432068, 0.6382832527160645, 0.4470524787902832, 0.26963791251182556, 0.6490927338600159, 1.7908964157104492, 0.9542490839958191, 1.5473016500473022, 0.8926109075546265, -0.49919524788856506, 0.2404126524925232, -0.3294731080532074, 0.5793271660804749, -2.1719870567321777, -1.2814213037490845], [0.5615072846412659, 1.0506742000579834, 0.616976261138916, 0.1995486617088318, 0.2874612510204315, -1.0578769445419312, 0.4787941873073578, 0.48708945512771606, 2.41495680809021, -0.15106990933418274, 0.20643320679664612, -1.2720704078674316, -0.41370394825935364, 1.723252296447754, 1.034469723701477, -0.7679234743118286, -1.4824614524841309, -0.5669673681259155, -1.3361129760742188, -0.0908636748790741, -0.771683931350708, -0.48010626435279846, 1.1295592784881592, 1.014573097229004, 0.4034918248653412, 0.012723609805107117, -0.28871530294418335, -1.5745793581008911, 0.771542489528656, -0.9551430344581604, 1.1573905944824219, -0.20590199530124664, -0.8619900941848755, -1.551652431488037, -0.08625559508800507, 0.18320263922214508, -0.8357205986976624, 0.32098889350891113, -1.1432071924209595, -0.23931919038295746, -1.038110613822937, -1.0761698484420776, -2.0281786918640137, -0.5812456011772156, -0.49089133739471436, -1.6559487581253052, -0.9409650564193726, -0.3548847436904907, -0.083468958735466, -0.41594111919403076, 0.3024732172489166, -0.40629255771636963, 0.3606663644313812, -0.5599465370178223, 0.8967660665512085, 0.04660810902714729, 0.593230664730072, -1.2872858047485352, 0.36387133598327637, 0.7544297575950623, -0.2802058756351471, -0.29058414697647095, 0.9549877047538757, -0.06009652465581894, -0.0913124829530716, -0.1830470710992813, -1.0039470195770264, 0.40551313757896423, -0.08211146295070648, -0.1140393540263176, 0.38634759187698364, 0.4484117925167084, 0.9149470329284668, -0.3391647934913635, -1.9764422178268433, -0.7307509183883667, -2.007197380065918, 0.5860897302627563, 0.5040369033813477, -0.3577090799808502, 0.04116043075919151, 1.8434652090072632, -0.7558480501174927, 1.845375895500183, 1.0093034505844116, 1.4997094869613647, -0.442462682723999, -0.26566046476364136, 0.32365426421165466, -0.7501962184906006, -0.32173997163772583, 0.09638041257858276, 1.1041882038116455, -0.5160609483718872, 1.1275441646575928, 0.999569833278656, 1.1928335428237915, 0.43789151310920715, -0.3485212028026581, -1.7851536273956299, 1.5655407905578613, 2.1658997535705566, -0.9908075332641602, 0.9290909171104431, -0.03126120567321777, -0.40476998686790466, -0.6166805624961853, -0.9380798935890198, -0.9542266130447388, -1.9340342283248901, -1.0280336141586304, 1.080325961112976, 0.029143622145056725, 0.5095189213752747, 0.5229045748710632, 0.38654306530952454, -0.2243587225675583, 0.7636846303939819, -1.8727750778198242, 1.513702154159546, -1.7484104633331299, 0.2376658320426941, -0.5930467844009399, 0.09369010478258133, 0.12602965533733368, 0.12208220362663269, 0.49931731820106506, 0.09826706349849701], [1.8674166202545166, -0.22533585131168365, -1.3261992931365967, -1.0870143175125122, 2.0019774436950684, 1.3651401996612549, 1.2893983125686646, 1.1983120441436768, 0.8457651138305664, -0.17556895315647125, -2.006157398223877, -0.2788422107696533, 0.13778872787952423, -1.210128664970398, -0.45136886835098267, 0.2686794400215149, 0.8950076699256897, 0.36237964034080505, -0.4167701005935669, -0.7194619178771973, -0.5004117488861084, 0.3598281741142273, -0.5133704543113708, 1.2133111953735352, -0.03808772191405296, -0.5262444019317627, 0.7518666982650757, 1.8231208324432373, 0.8156318664550781, 1.9297796487808228, -0.8524359464645386, 0.9837028980255127, -0.13536368310451508, 0.16202247142791748, 0.39769676327705383, 0.4543939232826233, 0.058097947388887405, 0.587812602519989, -1.1595523357391357, 1.3584808111190796, -0.5639973878860474, 1.874336838722229, 1.0730253458023071, 0.9151538014411926, -1.1424559354782104, 0.2652980387210846, -0.43956002593040466, -0.04837587848305702, 0.8148854374885559, -0.2797101140022278, -0.5091763138771057, 0.5058574080467224, -0.6845916509628296, 1.7884900569915771, -0.7948772311210632, -0.7124480605125427, 1.282702088356018, -1.315290093421936, 0.29505398869514465, 0.11587046831846237, -0.6176059246063232, -0.46887874603271484, 1.2370978593826294, 0.568032443523407, 0.451171338558197, 0.20944294333457947, 1.5968154668807983, -0.7306729555130005, 1.2216627597808838, -1.2162694931030273, -0.403584361076355, 1.0008262395858765, -0.4163056015968323, -1.7417691946029663, -1.320133924484253, 0.7012161016464233, 1.7886865139007568, -0.3572099208831787, 0.24369312822818756, -0.35990604758262634, -0.6031831502914429, -0.47618231177330017, -0.8885424137115479, 0.8206195831298828, 0.027197957038879395, 0.8795498609542847, 0.5388681292533875, 0.27854394912719727, -0.5852687954902649, 0.36293697357177734, -1.0754992961883545, 0.29467445611953735, 1.3490186929702759, 0.36832934617996216, 0.9257521629333496, 0.4600732922554016, -0.04106755182147026, -0.17316092550754547, -0.615425705909729, -1.4750162363052368, 1.5251964330673218, 1.8962230682373047, 0.2738991677761078, 1.764926791191101, 1.133993148803711, 2.0109753608703613, -0.5217019319534302, 0.2727113664150238, -1.3598158359527588, -0.440525621175766, 0.331449419260025, 1.7588895559310913, 1.284234642982483, -0.01067412830889225, -1.752625823020935, -0.21818557381629944, 0.43758368492126465, -0.03908411040902138, -0.41369348764419556, -0.6987738013267517, -0.5749267935752869, 0.8254644870758057, 0.6590964794158936, -0.6248636841773987, 0.18980278074741364, -1.8178014755249023, -1.4887983798980713, -0.994187593460083], [-0.27628999948501587, -0.7961910963058472, -0.12020129710435867, 0.19990979135036469, 0.3747826814651489, 0.6524173617362976, -0.6890771389007568, 1.3524951934814453, 1.5988367795944214, -0.2855072021484375, 1.1668018102645874, 2.7924575805664062, -0.6219186186790466, 1.3101955652236938, 0.6872367262840271, 0.09198088943958282, 0.5244566798210144, 1.6913915872573853, 1.3298290967941284, 1.2138442993164062, 0.2616356909275055, 0.02207142300903797, -0.3052167594432831, -1.7247320413589478, -0.3486361801624298, 2.2276413440704346, 0.17484666407108307, 0.15108969807624817, 1.2502468824386597, -0.40184691548347473, -0.6520903706550598, -0.5154424905776978, -1.002848505973816, 0.609235942363739, -0.21578925848007202, 1.58964204788208, -0.5285114645957947, -0.3394679129123688, 0.45611363649368286, 2.4246842861175537, 0.4846579432487488, 0.1480547934770584, -0.746086597442627, 0.5964305400848389, -0.5521214604377747, -1.8099726438522339, 0.7592115998268127, 0.619357705116272, 0.5813621878623962, -0.22353887557983398, -0.8453269600868225, 0.7752046585083008, -0.14295843243598938, 0.3408255875110626, -1.0857959985733032, -1.0407135486602783, 0.4848233163356781, 0.2379191517829895, 1.0825741291046143, 2.327531576156616, 1.0070104598999023, 0.5835101008415222, -0.24512438476085663, 0.8459981679916382, 0.7316925525665283, -1.3141604661941528, -2.351661443710327, 1.7185609340667725, 0.23811325430870056, -0.11175359785556793, -2.3483469486236572, -0.16462761163711548, -0.3771873712539673, -0.5233919024467468, 0.12093844264745712, 0.11815208941698074, -0.9851900935173035, -1.8628202676773071, 0.8326918482780457, -0.3818012773990631, -1.7791696786880493, 0.517676055431366, -0.2731570303440094, -0.9747467041015625, 0.03839637711644173, -0.3661532402038574, 0.150231271982193, -0.39342668652534485, -1.3193490505218506, 0.3467981815338135, -0.05525738373398781, 0.8859076499938965, -0.052485834807157516, 0.3765728771686554, -0.9110355377197266, 0.18174642324447632, 0.6744256615638733, 0.6891621947288513, 0.9181241989135742, 0.3522435128688812, 1.3237924575805664, 0.7265146970748901, 0.7409180402755737, 0.9746659994125366, 0.06029487028717995, -0.635033130645752, 1.104320764541626, 0.06937651336193085, 1.93808913230896, 0.1967795044183731, 0.12757596373558044, -0.09254901856184006, 0.5434157252311707, 1.266481876373291, -0.2664099335670471, -0.033606529235839844, 0.16834892332553864, 0.5836961269378662, -1.0084607601165771, 0.4577134847640991, 1.1101421117782593, -1.6365755796432495, -0.28717589378356934, -1.068824052810669, 0.38717207312583923, 0.41294044256210327, 0.9123286604881287, -0.20502343773841858], [-0.15722261369228363, -0.04678166285157204, -0.2798948287963867, -0.6323278546333313, 0.3437705636024475, -1.3836613893508911, -0.18300123512744904, 1.036324143409729, 0.5237505435943604, -1.0936681032180786, -1.2391886711120605, -0.9769372344017029, -0.3350508511066437, -0.9822789430618286, 0.9259611964225769, -0.3296446204185486, -1.6209757328033447, 0.05070750042796135, -1.1318016052246094, -0.09398680180311203, -0.03431856632232666, 0.7001729011535645, -0.20619598031044006, 0.1970236748456955, 0.17330989241600037, 1.537726640701294, -0.35007786750793457, -0.8581589460372925, 1.7419989109039307, -0.10246320068836212, -0.7706969976425171, 1.4485520124435425, 0.5140939950942993, -0.09182365983724594, 1.6478686332702637, 2.115093231201172, 1.8311376571655273, 1.4463140964508057, 2.754473924636841, 0.795714259147644, -0.932636022567749, -1.534851312637329, -0.3011370599269867, 0.24525360763072968, 0.9801250100135803, -1.2479422092437744, -1.0527724027633667, 1.959902286529541, -0.7818090915679932, 0.3282735049724579, 1.4945785999298096, 1.2865171432495117, -1.2737699747085571, 0.12874312698841095, 0.09164770692586899, 0.7816261053085327, -0.20127660036087036, 0.34547555446624756, 0.44788554310798645, 1.5201137065887451, -1.0721725225448608, 0.5463334918022156, 1.5692273378372192, -1.7281018495559692, 1.4454160928726196, -1.8904672861099243, -2.2027225494384766, -0.5790961980819702, 0.9263343811035156, 2.6567320823669434, -1.0594522953033447, -0.661349356174469, 0.09104105085134506, 0.5958113074302673, -1.5860121250152588, -0.7169954776763916, -0.13794848322868347, 0.2373836487531662, -1.049792766571045, -0.47799691557884216, -0.5354458689689636, 0.5003564953804016, -0.709123432636261, -0.1431157886981964, 0.1997690498828888, 1.194509506225586, -0.2739713788032532, 0.056617625057697296, -0.10799300670623779, -0.3286769986152649, 1.3911428451538086, 0.40134334564208984, 0.11178819835186005, 0.32494691014289856, -1.199664831161499, 0.3487158417701721, -1.0215892791748047, -2.004690408706665, -0.919614851474762, 0.43555504083633423, 0.5701364278793335, 1.0439521074295044, -0.6731203198432922, -2.08138108253479, 0.8772917985916138, -0.43574169278144836, -1.8790035247802734, 0.8363485336303711, 1.3154222965240479, -0.109518863260746, 0.029937580227851868, -0.3813423812389374, 1.0729517936706543, -0.13752123713493347, 0.517368733882904, 0.18206080794334412, -0.8391835689544678, 0.7187929749488831, 0.04928913339972496, 0.4686662554740906, -1.2045129537582397, -0.9943112730979919, -0.9960875511169434, 1.8899412155151367, 0.25669029355049133, -0.17811988294124603, 1.245408535003662, 1.0168166160583496], [1.1081753969192505, 0.6087381839752197, -1.0822752714157104, -1.5412359237670898, 0.2508206367492676, 0.27184978127479553, 0.20148149132728577, 0.44436517357826233, 0.7782109975814819, -0.07959269732236862, -0.4333135187625885, -1.4703166484832764, 0.11255152523517609, -1.2321674823760986, 1.081112027168274, 0.1996174305677414, 2.6697707176208496, 0.7014132142066956, 0.3216812014579773, -0.45333579182624817, 0.12472566962242126, -0.36998462677001953, 0.2567902207374573, -0.2508808970451355, 0.02057466097176075, -1.0614434480667114, 0.590806245803833, 0.889254629611969, -0.04233599826693535, 0.8969994783401489, -2.853726387023926, -0.4735185503959656, -0.7995765209197998, -1.3605108261108398, -0.7517557740211487, 0.3674510419368744, 0.3792896866798401, -2.756760835647583, -0.22530926764011383, 1.376479983329773, -0.8460785150527954, -0.05485072359442711, 0.16164293885231018, -1.944946527481079, -0.05429009348154068, -0.5753857493400574, -0.12444792687892914, -0.525462806224823, 0.5335217118263245, -1.0326628684997559, 0.857361376285553, 0.15452592074871063, -0.036416009068489075, 1.0776145458221436, -0.3412681221961975, 1.8177027702331543, 0.6710518002510071, 0.3164213299751282, 0.5725012421607971, -0.8406814336776733, 0.1821167916059494, -0.04041912034153938, -0.6646831035614014, 0.8813881874084473, 0.8332544565200806, 0.7048474550247192, -0.049139752984046936, 0.8125953674316406, 0.21025866270065308, 1.237254023551941, -0.20912709832191467, 0.6342015862464905, 0.11392059922218323, -1.0469865798950195, 0.5386143922805786, -0.9492495059967041, -0.8006730079650879, -0.39609119296073914, -0.6078916192054749, -0.8934920430183411, 0.6199566721916199, 1.084983229637146, 0.5772305130958557, -0.2013855278491974, -1.83936607837677, -0.39279115200042725, 0.0047060176730155945, 1.011144995689392, -1.4768304824829102, 0.5998603105545044, 0.18859830498695374, -0.7245950698852539, 0.9539254903793335, 0.3010135293006897, -0.730389416217804, 1.2178716659545898, 1.2827672958374023, 0.3830888569355011, 0.527538001537323, 1.0644817352294922, -0.3947075307369232, 0.0169195719063282, 0.864059567451477, -1.1111537218093872, -0.5707994699478149, 0.8250027894973755, -1.0374953746795654, -0.1422681212425232, 0.9872326254844666, 0.6065430641174316, 2.1273388862609863, 0.084109827876091, 1.2356139421463013, 0.02122057043015957, 0.3529229164123535, -0.40642988681793213, -0.6111922264099121, -1.2562731504440308, -1.6879280805587769, 0.9526259899139404, -0.1274711936712265, 1.5329519510269165, -0.4370756149291992, -0.49727892875671387, 0.0981450080871582, 0.19712218642234802, 1.1641995906829834, 1.2583723068237305], [0.4222254455089569, -0.7313489317893982, -1.1817700862884521, -2.790705442428589, -1.4724516868591309, 1.1885621547698975, -0.04005822911858559, 1.0466123819351196, -0.1659586876630783, -2.318033456802368, 1.505450963973999, -0.6653311252593994, -2.7010860443115234, -2.2032413482666016, 0.7712367177009583, -0.034704580903053284, -0.8494577407836914, 0.3806450068950653, -0.9700666069984436, -1.73308265209198, -0.15836773812770844, -0.49896079301834106, -0.34098631143569946, 0.48797786235809326, 0.513572633266449, 1.0019352436065674, -0.8447735905647278, 1.1038463115692139, -0.6576799154281616, 0.14474132657051086, -1.1112867593765259, 1.2168757915496826, 0.07101322710514069, -0.24141396582126617, 0.011013721115887165, -0.6020364761352539, -0.8101159334182739, -0.30549415946006775, 1.008485198020935, -0.798118531703949, 0.04141007736325264, 1.8646843433380127, -1.7040636539459229, 0.1868850290775299, 0.12198709696531296, -0.5776477456092834, 1.1411840915679932, -1.1031168699264526, -1.687846064567566, -0.796815812587738, -0.052970364689826965, -1.1567211151123047, -0.29214704036712646, 2.2259411811828613, -0.2251804769039154, 2.604926109313965, 2.886641502380371, -0.8913339972496033, -0.1405477225780487, -1.9199520349502563, 0.7076051831245422, 0.26800140738487244, 0.7495595812797546, 0.11085251718759537, -0.46535882353782654, -0.5696507096290588, -0.2923934757709503, -0.7547443509101868, 0.12592460215091705, 0.3058832287788391, -0.22127950191497803, -0.7751023769378662, -0.4888516962528229, 0.3308160603046417, 0.0031138285994529724, 0.22596202790737152, -0.26501330733299255, -0.6650277972221375, 0.5477136969566345, -1.1732407808303833, -0.1412285417318344, -0.46323469281196594, -0.12327373772859573, 0.037546806037425995, 1.8529119491577148, -0.383835107088089, -0.4007154405117035, 1.2329442501068115, 0.9493907690048218, 0.8518105745315552, 0.911828875541687, 1.324806571006775, -0.4603813588619232, -1.0092346668243408, 0.412416011095047, -1.082698941230774, -0.8071816563606262, 0.8550775647163391, 1.9876657724380493, 0.21921677887439728, -0.06524468958377838, -1.913764238357544, -0.43762490153312683, 0.2048855721950531, -0.25441640615463257, 0.4446212947368622, 1.0880972146987915, -0.7955989837646484, 0.3393387794494629, 0.12000448256731033, 0.18608945608139038, -0.4580870270729065, -1.1008341312408447, 1.2943861484527588, 0.037757668644189835, -0.061820998787879944, 0.6351805329322815, 0.1722838133573532, -0.3982310891151428, 1.5650250911712646, -1.5031548738479614, -0.4246228039264679, 0.20873118937015533, 2.222907066345215, -0.2845737636089325, 0.27180349826812744, 0.98272305727005, -1.2921173572540283]]]], "dtype": "float32"}, {"__nd__": [[[[-1.0117104053497314, 0.9335201382637024, -0.451303094625473, -1.2263296842575073, 0.7632288336753845, -0.8278417587280273, 0.010565963573753834, 1.6200591325759888, 0.6582101583480835, -0.30094197392463684, -0.04913479834794998, -1.323626160621643, 0.26711031794548035, -1.0938215255737305, -0.44224056601524353, -0.7307794094085693, -0.11868131160736084, -0.25117847323417664, -0.9532260298728943, 1.4012197256088257, -1.367928385734558, 0.46631526947021484, -0.11269596219062805, 0.7227606773376465, 0.5273635387420654, -0.3858319818973541, 0.07628729939460754, -0.3612438142299652, -0.5287783145904541, -0.20330001413822174, -1.3272496461868286, 0.4545709490776062, -0.8892456889152527, 0.1719239205121994, -1.7149739265441895, 0.3349698781967163, 0.6627826690673828, 0.5683130025863647, 0.759637176990509, -0.13745249807834625, -0.03627949580550194, 0.16453932225704193, -0.4416479766368866, -1.8035930395126343, 1.197482705116272, -0.29650524258613586, -0.28766897320747375, 0.1917944699525833, -0.7285622358322144, 0.7734291553497314, 0.2673848271369934, 0.37748003005981445, -0.4371030926704407, 0.018593881279230118, -0.7673671245574951, 1.0099951028823853, -0.28341802954673767, -1.6518536806106567, -0.2695118486881256, 0.6850807070732117, -0.0049775647930800915, -0.3721587061882019, -2.0886502265930176, -2.98928165435791, 0.590541660785675, -0.45414769649505615, 0.39861392974853516, 3.171799421310425, 1.198190450668335, 1.2567176818847656, 0.8465509414672852, -0.6947256922721863, 0.2733912467956543, 0.41775596141815186, -0.37389782071113586, -0.48544371128082275, 0.08516072481870651, -0.2691684067249298, -0.012329933233559132, -0.0838053748011589, 0.9727767109870911, -0.9108471274375916, 0.32201796770095825, -1.2192022800445557, 0.7231977581977844, 0.05641915649175644, -1.112402319908142, -0.7803781032562256, -0.4110848307609558, 1.0128408670425415, 0.7528330087661743, -0.40997061133384705, 0.946800708770752, -0.27734634280204773, -1.2780686616897583, -1.1184232234954834, 0.7246096134185791, -0.9172729253768921, 0.01093009952455759, 0.6286709308624268, 0.15824581682682037, 1.9329240322113037, -0.5247431993484497, 1.7385588884353638, 0.5013198256492615, 1.6691179275512695, -1.0350841283798218, 0.10984718054533005, 0.8824743628501892, 0.5146358609199524, -0.64913010597229, -0.34961625933647156, 0.2713601887226105, -1.4872784614562988, -1.5476309061050415, -0.35032886266708374, 0.5765484571456909, 1.4570873975753784, -0.35373663902282715, -0.2548862397670746, -2.790790557861328, -0.5438397526741028, -0.03623490035533905, -1.1118738651275635, -1.942761778831482, -0.5037588477134705, 0.9223284125328064, 0.30002522468566895], [0.8466194868087769, 1.837976336479187, -1.0936403274536133, 1.186095118522644, -0.43726298213005066, -0.2130466252565384, 0.8238608837127686, 0.24688006937503815, -0.7064855694770813, 0.18285442888736725, -0.6027823686599731, -0.04057598114013672, -1.337611436843872, 0.15097588300704956, 0.4274248480796814, -0.45982998609542847, 0.10836411267518997, 1.959058165550232, 0.7264293432235718, 0.23089636862277985, -1.4490660429000854, 1.2078914642333984, -0.8091011643409729, -0.9463975429534912, 1.1813623905181885, 0.5295292735099792, 1.2962137460708618, 1.2003616094589233, 0.712067723274231, -0.6832323670387268, -0.3386135697364807, -0.34866002202033997, -0.6478906273841858, 0.86167973279953, 0.9409284591674805, 0.9189414978027344, 1.479127049446106, 1.423805832862854, 0.40872707962989807, 0.9446740746498108, -1.0025047063827515, 0.8312131762504578, 0.4130174517631531, 2.0538899898529053, -0.2817881107330322, -0.7700937986373901, 0.09497398883104324, 2.0595288276672363, -0.0946805328130722, -0.2824881672859192, 1.491692304611206, 1.0811902284622192, -0.14841504395008087, 0.8633678555488586, -0.6912017464637756, -1.4573029279708862, -0.31533703207969666, -0.7238425016403198, -1.5727020502090454, -0.904904842376709, -0.29384908080101013, -1.0499144792556763, 1.994815468788147, 1.0931352376937866, 0.037446264177560806, -0.19878822565078735, 0.04901763051748276, 1.5153741836547852, 1.0528361797332764, -1.2926771640777588, -0.7853744626045227, -0.7618537545204163, 0.6468906402587891, 0.2838287949562073, 0.0732516497373581, -0.43803641200065613, 1.304041862487793, -1.5466631650924683, -1.0884100198745728, -1.3938994407653809, -0.6544229388237, 0.46389347314834595, 0.03734970465302467, 0.8913193941116333, -1.2364615201950073, 0.5126781463623047, -0.07636549323797226, 1.819471836090088, 0.2580806612968445, -1.2232513427734375, 1.3963661193847656, 0.07148782908916473, -0.5632786154747009, -0.09295166283845901, 0.3826117515563965, 0.27891266345977783, 1.100400686264038, -0.1516876071691513, 0.9120253324508667, 1.1342058181762695, -0.762453019618988, 1.610018014907837, -1.145342469215393, -0.91465824842453, -0.5562298893928528, -0.15968072414398193, -1.0607752799987793, 0.7296485304832458, -1.1046181917190552, 0.8085893988609314, -0.6672510504722595, -0.7173900008201599, 0.23976659774780273, -1.6041277647018433, 0.8752736449241638, -0.8469345569610596, -2.6540095806121826, 1.5374470949172974, 1.742480754852295, 0.5955769419670105, 0.07814965397119522, 0.1025855615735054, 2.094060182571411, 0.13152393698692322, -0.8668789267539978, -0.21166178584098816, -0.5998049974441528, 0.4206589460372925], [0.2856854498386383, 0.05071140453219414, -0.6887101531028748, -2.1497385501861572, 1.3408854007720947, -1.181209683418274, -0.7258595824241638, 1.6963249444961548, 1.4162616729736328, 0.6308690905570984, -1.088204264640808, 0.7456018328666687, 0.4510505795478821, 0.5370715856552124, -0.8429857492446899, 0.4912719130516052, 1.8146001100540161, 0.3583469092845917, -1.217572569847107, -0.15162639319896698, -0.07977811247110367, -0.714358389377594, 0.7699984908103943, -0.8419811129570007, 0.1351512372493744, 0.3516247272491455, -0.8753753900527954, -0.16500699520111084, -0.8778193593025208, 0.9296826124191284, -1.1119939088821411, -0.9045670032501221, 1.2776353359222412, -0.37852343916893005, -1.5635076761245728, -1.392924189567566, -1.524101734161377, -1.6344385147094727, 0.33705630898475647, 0.29792097210884094, -0.03453169763088226, 0.8286817669868469, 0.6723463535308838, 1.3110742568969727, -0.44420742988586426, -0.38614287972450256, -0.9955267906188965, 0.15872594714164734, 1.062048077583313, -0.8140767216682434, -1.6508748531341553, -0.35159918665885925, 2.3982763290405273, 0.012807314284145832, 0.991329550743103, -1.40389084815979, -0.4310337007045746, -0.7273237705230713, 1.4068289995193481, -0.01686880551278591, 0.4487609267234802, 0.6235225200653076, 0.6730425953865051, 2.462475299835205, 0.06664825975894928, 1.4102622270584106, -0.38699668645858765, -0.5022294521331787, 1.310518741607666, -0.36935240030288696, 2.3903093338012695, 0.41021302342414856, 1.0178602933883667, 0.6731606125831604, 0.47434887290000916, -1.1069817543029785, 0.5594871640205383, -0.22823002934455872, 0.5266968011856079, 0.29911133646965027, -0.44388049840927124, 1.1183544397354126, -0.7043244242668152, -0.6486608386039734, 0.6506369113922119, -1.0473737716674805, 0.8643016815185547, -0.5531091094017029, 0.053894154727458954, 0.8734946846961975, -0.332968533039093, 0.7437638640403748, 1.592299461364746, 0.6927591562271118, 0.7656265497207642, 0.798627495765686, 0.9438679218292236, 0.026805082336068153, 0.1415417641401291, 0.1929139792919159, -0.32588180899620056, -0.24001848697662354, 1.0471343994140625, 2.3487343788146973, -0.02717159129679203, -0.9078903198242188, 0.26602447032928467, 0.6298001408576965, 0.7203779816627502, 1.04121994972229, 1.5362221002578735, 0.297800749540329, 0.6785626411437988, -1.0362954139709473, 0.1119743287563324, -0.9762356877326965, 0.2894684374332428, 0.7582279443740845, -0.5742383599281311, 1.0886303186416626, -0.35830116271972656, 0.8277290463447571, -0.32417601346969604, -1.6489723920822144, -0.10329452157020569, -0.058557238429784775, 0.5365780591964722, 1.0164754390716553], [-0.07797982543706894, -0.41127800941467285, -1.5129903554916382, -0.9888431429862976, -0.2883751690387726, 1.8545739650726318, -1.4564180374145508, -0.556085467338562, -1.4943184852600098, -0.0195546243339777, 0.7546696066856384, 0.782127320766449, -1.2342026233673096, -1.218906044960022, 3.0307869911193848, 0.7679125070571899, 0.28479933738708496, -1.2959892749786377, 1.4078463315963745, -0.5348013639450073, 1.0142673254013062, 1.3335999250411987, 1.4938855171203613, 0.1626851111650467, -0.2741263210773468, 2.403485059738159, -2.3079659938812256, -1.0379018783569336, 0.4493095278739929, -0.28762125968933105, 0.6085277199745178, 1.1704086065292358, -0.6060412526130676, 1.2252404689788818, -0.922468364238739, 0.5090239644050598, 0.830793023109436, -1.5208985805511475, 1.2523770332336426, -1.661368489265442, -0.1743556708097458, 0.49141764640808105, 0.08879641443490982, 0.5929961204528809, -0.5184958577156067, 1.4586265087127686, -0.3042157292366028, 0.14470431208610535, 0.09785294532775879, -0.4642152786254883, 2.457961082458496, -0.12101173400878906, 1.0778305530548096, -1.3861455917358398, -1.4464200735092163, -0.5358512997627258, 0.055431582033634186, -2.25765323638916, 1.2560864686965942, -0.25999101996421814, 0.5778257250785828, 0.30274373292922974, -0.42593157291412354, -1.4652413129806519, -0.04274553433060646, 0.36536964774131775, 0.812351405620575, 0.22951550781726837, -1.11231529712677, -1.7172681093215942, 1.8428438901901245, 0.621620774269104, -1.6643574237823486, -0.8913230895996094, -1.6921660900115967, -1.2343891859054565, 0.1931341290473938, 1.0167423486709595, -0.9966773390769958, -1.068220615386963, 0.17386575043201447, 0.20424099266529083, 0.0747576653957367, -0.16138023138046265, 0.8059946894645691, 2.3291032314300537, 0.45804449915885925, 0.38324642181396484, 2.017130136489868, -0.7105146050453186, -0.022341979667544365, 0.1187799721956253, -0.7394886016845703, 0.9935998916625977, -1.4895075559616089, 2.2439985275268555, 0.06183220073580742, -0.11257676780223846, 0.3340550363063812, -0.1793413907289505, 1.0334222316741943, -2.451991319656372, -1.3211302757263184, -0.9851635694503784, 0.4920543134212494, -0.269768625497818, -0.44381800293922424, 0.023036764934659004, 0.09482858330011368, -0.14400146901607513, -0.3412332534790039, 0.5030616521835327, -0.530767023563385, 0.20535734295845032, 1.0823063850402832, -1.2452865839004517, 0.38807058334350586, 0.6909646391868591, 1.2120299339294434, -0.5622862577438354, -0.5392406582832336, 1.746989130973816, -0.13685716688632965, 0.1927214413881302, 1.9493886232376099, 1.6548542976379395, -1.5157794952392578, -0.1861904114484787], [1.8853377103805542, -1.2012594938278198, 0.8261412382125854, -0.5282706618309021, 1.3711220026016235, -0.12958484888076782, 1.617324709892273, 0.09254787862300873, -0.34882041811943054, 0.5640183687210083, -1.3888200521469116, -1.2460464239120483, -0.4202330708503723, 0.2756313383579254, -2.2440760135650635, 0.33327752351760864, 0.3933502435684204, -0.6429456472396851, -0.49272021651268005, -0.2693130075931549, -0.9472631812095642, -0.2602522373199463, 0.6808265447616577, -1.601565957069397, -0.3149987459182739, -0.9245532155036926, 0.2590944766998291, 1.367648959159851, 0.8084326982498169, 0.061910081654787064, -0.925971269607544, 0.7461667656898499, 1.7397068738937378, 0.18773910403251648, 1.5723276138305664, 0.23367749154567719, -0.6170943975448608, 1.5796043872833252, -1.319560170173645, 0.30109450221061707, -0.18356989324092865, 2.071141242980957, 2.265784740447998, 0.4689510464668274, -0.7935059070587158, -0.4967247247695923, 0.24261732399463654, -2.032109022140503, 0.14600060880184174, 0.23083433508872986, -0.6572570204734802, 0.13915185630321503, 0.6022141575813293, 0.4153164029121399, -1.7650675773620605, 2.3068532943725586, 1.2740705013275146, 0.3331669867038727, -0.09908828139305115, 0.28446581959724426, -0.6993299722671509, -0.7570850253105164, 0.05732329189777374, -1.0516520738601685, -0.8478567004203796, -0.7824109792709351, 1.0541982650756836, -0.03311124071478844, 1.1613283157348633, 0.3628154695034027, -1.85875403881073, -0.0021003324072808027, -0.6662813425064087, -0.1521724909543991, -0.22112765908241272, 0.06945192813873291, -0.25237730145454407, 0.6120046973228455, -1.032866358757019, 0.24794518947601318, 2.423092842102051, 0.6024203300476074, 0.12509119510650635, -0.5076801180839539, -0.3827034533023834, -1.2746859788894653, 0.7164760828018188, -0.5368106961250305, -0.6432360410690308, 1.157813549041748, 1.2417980432510376, -0.8707187175750732, 0.8138786554336548, 0.12350344657897949, -0.3614998459815979, -0.31184157729148865, 0.7435952425003052, 0.5122065544128418, -0.2669404447078705, 0.8456189036369324, -1.179919958114624, -0.490043580532074, 0.3603043556213379, -1.3273394107818604, 0.26389285922050476, -0.45120081305503845, -0.6514290571212769, -0.3941807150840759, -0.5052582025527954, 0.5733169913291931, -2.2660038471221924, -1.843817114830017, 0.5208407044410706, -0.05011036992073059, 0.744415819644928, -1.3821971416473389, -0.8885790705680847, 1.031794548034668, 0.3571576178073883, -0.48883697390556335, 0.4463815689086914, 0.3476526439189911, -1.2772780656814575, 1.106459617614746, -2.268436908721924, 0.3639530539512634, 0.7990981936454773, -1.090627908706665], [1.8032236099243164, 0.2544737458229065, -0.3554230332374573, -0.13660620152950287, 0.5649319291114807, 0.5762772560119629, 0.7987431287765503, 0.978918731212616, 0.5276716351509094, -0.3982802629470825, 0.3398520052433014, -1.77336585521698, -0.740542471408844, -1.9305214881896973, -0.7912283539772034, -1.323599100112915, 0.22102630138397217, 0.8081075549125671, -0.32193315029144287, 0.6017892956733704, -0.35464373230934143, -0.6168015599250793, -0.42883527278900146, 1.318687915802002, -0.5116541981697083, 0.6520110368728638, -0.230460062623024, -0.5445789694786072, -1.0459831953048706, 0.41023752093315125, -0.6373578310012817, -0.45319831371307373, 1.1564987897872925, -1.2016935348510742, 1.905350685119629, 1.3646721839904785, 0.8701295852661133, 0.6534951329231262, -0.7161033153533936, 0.6854454278945923, -0.0661332830786705, -0.07840269058942795, 0.028128093108534813, -0.02944895066320896, 1.4433691501617432, 0.7118316292762756, -0.24862231314182281, -0.05085577443242073, 0.6527724266052246, -0.4333449602127075, 0.3726968765258789, 1.5528908967971802, 0.7644163370132446, 0.4632473587989807, 0.09763088822364807, -1.0551657676696777, 0.5031464695930481, -3.560127879609354e-05, -0.18617352843284607, -0.30236145853996277, -0.45831435918807983, -0.4253408908843994, 0.3734707832336426, 1.665500521659851, -0.6343549489974976, 0.7988545298576355, 1.4353004693984985, -0.7386101484298706, -0.25519731640815735, -0.9199848771095276, -1.7190594673156738, -1.149214267730713, 0.2948075234889984, 0.5340098738670349, -0.434153288602829, 0.7583251595497131, 1.333317518234253, 0.32266074419021606, 0.7350070476531982, -0.3693859279155731, 0.6511572599411011, -0.6937059760093689, 0.20120292901992798, 0.1215607225894928, -0.021326478570699692, -1.1709808111190796, 0.21275882422924042, 0.9297031760215759, 0.1884002834558487, 1.2935807704925537, -0.474338561296463, -0.543037474155426, -1.7184864282608032, 0.10769762843847275, -1.0129300355911255, -0.26574432849884033, 0.886603057384491, -0.566214382648468, -1.358778476715088, 1.5010639429092407, -1.3501514196395874, 0.9796149730682373, 0.26234862208366394, -0.6665881872177124, 0.3977722227573395, -1.1229785680770874, -0.9261993169784546, 1.050491213798523, 0.6243250370025635, 1.3504555225372314, 0.36016204953193665, -0.6819942593574524, -2.1080102920532227, -0.16448989510536194, -0.821456253528595, 0.5627307295799255, -0.6360304951667786, 0.052239157259464264, 0.29718393087387085, -1.3564459085464478, -0.16334058344364166, -1.456514835357666, -1.037514567375183, -1.2514504194259644, -1.3907479047775269, 0.4796012043952942, -1.8461180925369263, -0.3422764837741852], [-0.06271567940711975, -1.1163872480392456, -0.1831607222557068, 1.346483588218689, 0.5736571550369263, 0.6891177296638489, 0.2559172511100769, -0.38923946022987366, -0.974170982837677, -0.4997549057006836, 1.3079760074615479, 0.6324865221977234, -1.2355245351791382, -0.43159255385398865, 1.4694647789001465, -0.298939049243927, 0.08781932294368744, -0.3147990107536316, -1.041386604309082, 0.09414997696876526, -1.0663427114486694, 0.7323859930038452, 0.5868753790855408, 1.7042251825332642, -0.5508595705032349, 0.09560307115316391, -0.2949802875518799, -0.4342944025993347, -1.8340486288070679, -0.14197500050067902, 0.6948485374450684, 0.9057608842849731, 0.7608606219291687, 0.8557769656181335, 1.123290777206421, 1.0174022912979126, 2.4890711307525635, 0.6597754955291748, 0.12921947240829468, -1.474287986755371, 0.6630179286003113, -1.3689578771591187, -0.5136245489120483, 0.8832647204399109, 0.15023694932460785, 1.3060951232910156, -0.8889542818069458, 0.5624812841415405, 0.0592656210064888, 0.31723034381866455, 2.004396915435791, -0.6100723743438721, 0.1640237718820572, -0.15211567282676697, 1.577896237373352, 0.49808230996131897, 1.8104335069656372, 0.5751250982284546, 1.16327965259552, 0.05741716921329498, -1.2168844938278198, 0.5641921162605286, 0.8759986758232117, -0.061748430132865906, 1.3815069198608398, -0.583489716053009, 0.38631483912467957, -0.6007159352302551, -0.5592127442359924, 1.1532602310180664, 0.9389171600341797, 0.1660727709531784, 0.9593021273612976, -1.7546061277389526, 2.0706379413604736, -0.011696415022015572, 0.24676251411437988, 0.7799571752548218, -0.10064487904310226, 1.0334969758987427, 1.2049903869628906, -0.49452823400497437, 2.1654090881347656, 0.6228927373886108, 1.1592934131622314, -1.1062135696411133, 1.0658711194992065, -0.5255699157714844, -2.562786817550659, -1.106654405593872, 0.7316757440567017, 0.5690377354621887, 0.7806364297866821, -0.34933337569236755, -0.5076390504837036, -0.0019255526131018996, 2.0861055850982666, -0.010357958264648914, -1.102824330329895, -0.2688847780227661, 1.554669976234436, 0.6130382418632507, 0.9835019707679749, 0.33150774240493774, 0.5391654968261719, -1.2613252401351929, 0.12982463836669922, -1.4368772506713867, -1.9981777667999268, 0.14088349044322968, 1.5154242515563965, 0.7037099599838257, -1.2991172075271606, -0.10484316945075989, 0.21640026569366455, 0.25197356939315796, -1.1794742345809937, 1.2676805257797241, -0.8856030106544495, -0.4104304313659668, -0.4203778803348541, -0.19320493936538696, 0.10004378855228424, 0.4726814925670624, -0.08039907366037369, 1.0810492038726807, 0.4400179386138916, 0.6165766716003418], [0.19053184986114502, -1.5282450914382935, -0.8691707849502563, -0.060893088579177856, -0.46598511934280396, -1.729119062423706, -0.6626101732254028, 0.057087820023298264, 0.7409507036209106, -1.6860324144363403, 1.396989107131958, 0.1482090801000595, -0.8215794563293457, 0.3941073715686798, 0.15005356073379517, 0.5610260367393494, 0.140967458486557, -1.6846754550933838, -0.2253793179988861, 0.977229118347168, 0.17128288745880127, 0.9089693427085876, 0.7391437888145447, -0.9311903715133667, 0.9031960368156433, -1.1003199815750122, -0.22198668122291565, -0.9928163290023804, 0.686059296131134, 0.3122022747993469, 2.222355842590332, -0.9724791646003723, 0.7640063166618347, -1.0091437101364136, -0.6324438452720642, 0.7465107440948486, 0.3471725583076477, -0.9177529215812683, -0.9589341282844543, 0.36657780408859253, -0.7154597043991089, -1.6951165199279785, 0.7380507588386536, -0.19211791455745697, -0.2559460401535034, -0.6617609262466431, -0.4900042712688446, -1.0109153985977173, -0.9424991607666016, 1.0488413572311401, -0.986659824848175, 0.8097102046012878, -1.711771011352539, -1.0792210102081299, -1.0261448621749878, -0.31423312425613403, -2.7887346744537354, 0.13945460319519043, -0.8875839710235596, 0.39895761013031006, 0.1833258867263794, 1.7755067348480225, 1.8351610898971558, 0.08826104551553726, -1.3520234823226929, -0.4072423577308655, 0.717126727104187, -0.2469986379146576, -1.0887889862060547, -1.3424828052520752, 1.9345016479492188, -0.453138142824173, -0.1543489247560501, 0.800418496131897, -1.407444715499878, 0.765064537525177, -0.6055191159248352, 0.5161124467849731, -1.692460298538208, 0.2588675022125244, -1.424371600151062, -0.8004181385040283, 0.10208673775196075, 0.42871153354644775, 1.1008641719818115, -1.6808592081069946, -0.09659594297409058, 1.8280794620513916, 0.5533767938613892, -0.7574319243431091, 0.5493625402450562, 2.1859638690948486, -1.4193379878997803, 0.6161081790924072, 1.2295269966125488, 0.5447848439216614, -1.4148645401000977, -0.032009147107601166, 0.48559215664863586, -1.1055693626403809, 0.7905207872390747, -0.9612642526626587, 1.3483322858810425, -0.6575922966003418, -1.1703267097473145, 0.491110622882843, -0.5673192143440247, 0.4493824243545532, -0.39592212438583374, 2.3763792514801025, 0.9001085162162781, 2.073080062866211, 0.97716224193573, 1.0819566249847412, -0.2397335022687912, 0.3201316297054291, -1.048094391822815, 0.7327507734298706, 1.7091952562332153, 0.9711753726005554, -0.06260482966899872, -0.16366133093833923, 0.7738052606582642, 0.08655380457639694, -1.572179913520813, 0.1558765321969986, 1.5553911924362183, 0.09610423445701599]], [[0.25917837023735046, 0.6996849179267883, -1.2588521242141724, 1.3160176277160645, 0.6729167103767395, -0.09385993331670761, 0.3888896405696869, 0.817793071269989, -0.25494545698165894, 0.3924514353275299, -1.007385492324829, -0.8120245337486267, -0.46242570877075195, -0.7253072261810303, 0.9716701507568359, 1.3778159618377686, 1.812731385231018, -0.8473528623580933, 1.022578477859497, 0.0693087950348854, 0.28906023502349854, -1.041996717453003, 0.32311320304870605, 0.6337449550628662, -0.16843648254871368, -0.4165940284729004, -0.4677470624446869, 0.08905594050884247, -0.9420294165611267, -1.0530563592910767, -1.9238543510437012, -0.7667838931083679, -1.5902854204177856, -0.171530619263649, 1.2740483283996582, -0.7058231830596924, -0.2020234614610672, 0.8251774907112122, -1.1358493566513062, -0.7679325342178345, -2.7200615406036377, -1.954530119895935, 1.8872215747833252, 0.4353104531764984, 0.8936071991920471, -0.5975635051727295, 0.1299053430557251, -0.5044389963150024, 0.595628559589386, 1.651653528213501, 0.4190027713775635, -1.5776311159133911, 1.3231606483459473, 0.15006029605865479, -1.5434722900390625, 1.8551440238952637, -0.8418397903442383, -0.39805975556373596, -1.0869179964065552, -0.7635732293128967, -0.5078961253166199, -0.5687612891197205, -0.7287503480911255, -1.7842789888381958, 1.4459028244018555, -0.6538563370704651, -0.42299333214759827, -0.5189059376716614, -0.7283146381378174, 0.8552499413490295, -0.005423910915851593, 0.8176307678222656, -0.001522925216704607, 0.44770124554634094, 0.630046546459198, 1.3540912866592407, -0.4229765832424164, -0.8372488617897034, -1.7892460823059082, -0.842587411403656, 0.8068380951881409, 1.3555954694747925, -2.0102550983428955, 1.3788584470748901, 0.04524063691496849, 1.1028691530227661, 0.3190414309501648, 0.3738431930541992, -0.7288128733634949, -1.052411675453186, 2.3682355880737305, -0.5931867361068726, -0.003774631302803755, 1.2856186628341675, -1.9494283199310303, 0.13452252745628357, -2.0098514556884766, -0.22257553040981293, -0.34195953607559204, 0.19755087792873383, 1.707353949546814, 1.3162055015563965, 1.8682198524475098, -0.3832051157951355, 0.6882277727127075, 0.9811802506446838, -0.825284481048584, 1.2099252939224243, 0.041562750935554504, -0.12298446893692017, 1.1022967100143433, 0.07618151605129242, 1.7165038585662842, 0.7448431849479675, -0.4685967266559601, -0.24047407507896423, 0.8400970697402954, 0.22224129736423492, -0.43761134147644043, -0.9070292711257935, -0.6916878819465637, 0.2114514857530594, 0.9870394468307495, -0.5631146430969238, 0.20273597538471222, -0.35651129484176636, -0.5099979639053345, 0.07424169778823853], [-0.3010962903499603, -0.46614667773246765, 0.4956408441066742, 0.27407288551330566, -1.3164693117141724, 0.19439755380153656, 0.3034282326698303, 0.5277153849601746, 0.46814751625061035, 0.08778703212738037, -1.7222414016723633, -0.3070010542869568, -0.7685633897781372, 0.2585074305534363, -0.3000117242336273, -0.675896942615509, 0.058989640325307846, 0.13684935867786407, 1.1471996307373047, -1.245078206062317, 0.405815452337265, 0.9855526685714722, 0.8192386627197266, -0.7133342027664185, -1.1442667245864868, 1.937623143196106, 1.1659502983093262, -1.6901021003723145, -0.5463205575942993, -1.159694790840149, -0.30515116453170776, 1.3154895305633545, 1.660302758216858, 0.737330436706543, 0.4607575833797455, 1.0555351972579956, -2.1979284286499023, 0.5409800410270691, 1.8091903924942017, 0.10920386761426926, -0.5311685800552368, -0.22754447162151337, 1.1605347394943237, 1.4366501569747925, 0.3791131377220154, -1.5560436248779297, 1.5243064165115356, 0.7288411259651184, 1.252277135848999, 0.08313728868961334, 0.9202455282211304, 0.6754865646362305, -1.749108910560608, -0.984633207321167, -1.0015864372253418, 1.536445140838623, -1.4398155212402344, -1.1941258907318115, 0.03981439396739006, 0.015414480119943619, -0.7440212368965149, -2.103327512741089, 1.577272653579712, 0.07873605191707611, 0.4503418803215027, -0.9309035539627075, -1.041459083557129, 0.4926770329475403, -0.43998003005981445, -0.3642413914203644, -0.9508169293403625, 0.7019251585006714, 0.05098960921168327, 0.9268694519996643, 0.16156211495399475, -0.5708790421485901, 2.1622540950775146, 0.678567111492157, 0.37471309304237366, 0.07653550803661346, 0.08653116226196289, 0.5101313591003418, -1.236433982849121, 1.3423662185668945, 0.08858874440193176, -1.7289584875106812, 0.056994710117578506, -0.5809314250946045, -0.3945527970790863, 2.2996344566345215, -1.506795883178711, -0.3692285716533661, 2.250277042388916, -0.32366377115249634, -1.8860348463058472, -0.28781750798225403, 0.4156707227230072, 0.25918954610824585, -0.7264434099197388, 0.25934141874313354, -0.20190294086933136, -0.6562328338623047, 0.6547191739082336, -0.6684677004814148, -0.2761015295982361, 1.6464943885803223, 0.2708318531513214, -0.6646249890327454, -0.4994516372680664, 0.6959385275840759, -0.6760463714599609, -0.054935991764068604, -1.0604888200759888, 0.9361985325813293, 1.2791368961334229, 0.5077879428863525, -1.2103315591812134, 0.3675819933414459, 2.0081779956817627, 0.8196552991867065, 0.3118114471435547, -0.3243366479873657, 0.08788908272981644, -0.0836975947022438, -0.0363430455327034, 0.28388914465904236, -0.5228292942047119, -2.0828161239624023], [-0.5888671278953552, -0.8729358911514282, -0.4420401453971863, 1.496018409729004, -0.7918063402175903, -0.10554894804954529, -0.17484213411808014, -1.3344038724899292, 1.701765775680542, 0.5259021520614624, 1.0764158964157104, 0.09203432500362396, 1.9953902959823608, 0.7917936444282532, 1.559985637664795, 0.1940263956785202, 0.2901041507720947, -0.1778186559677124, 0.3538782596588135, 1.167150855064392, 1.216300368309021, -0.4124109745025635, 0.5275922417640686, -1.6637458801269531, -0.1634204387664795, -0.4843951165676117, -1.5586225986480713, -0.4401068091392517, -0.4073527753353119, -0.13043440878391266, 2.2948808670043945, -0.6675163507461548, -1.2124642133712769, -1.3294360637664795, -1.3262478113174438, -0.6641072034835815, 1.6850780248641968, -0.14632466435432434, 0.26687026023864746, 0.802559494972229, 0.41662630438804626, 1.1845704317092896, -0.35819774866104126, 0.1701115220785141, -0.22882746160030365, 0.6212381720542908, 1.2880555391311646, 1.5538246631622314, 0.24824807047843933, -0.8934218883514404, 1.1223924160003662, 0.5148742198944092, -0.009095367975533009, 0.6478602290153503, -2.0692851543426514, -0.07209726423025131, 0.24255990982055664, 0.2509213089942932, -0.666151762008667, -0.5246931314468384, -0.371567964553833, 1.0374728441238403, -0.7182652950286865, -0.22505757212638855, 1.4948738813400269, -0.5486703515052795, 0.4892028570175171, 0.365203857421875, 1.5507398843765259, 0.006544528063386679, -0.1882414072751999, -2.4000351428985596, 0.3263236880302429, 0.1476754993200302, -0.4317486882209778, 0.6981935501098633, -0.3769916296005249, -1.172091007232666, -2.982684850692749, 0.2588653564453125, 0.36746740341186523, -0.28655770421028137, 0.4454348087310791, -0.7792149186134338, -0.14651724696159363, -0.5579913854598999, -1.3069666624069214, -0.6649685502052307, -0.11009860783815384, -1.7592922449111938, 0.25048476457595825, 0.3026067018508911, -0.22010032832622528, -0.7590686082839966, -0.001821076381020248, -0.08339624851942062, -0.5495213270187378, -0.07597128301858902, -0.2250688076019287, -1.2130180597305298, 1.1001142263412476, -1.8404234647750854, -0.490162193775177, 0.03929832577705383, -0.127357617020607, 0.1565132588148117, -2.0096161365509033, -0.1839761584997177, 1.4329277276992798, 0.18319813907146454, -0.4397083818912506, -0.23508985340595245, 0.5133289098739624, 0.5275806784629822, 0.8895952701568604, -2.2354485988616943, 1.5612884759902954, 0.007694180589169264, -0.7139508724212646, -1.723260521888733, 0.227274090051651, -1.0063148736953735, -1.214538812637329, -1.0912752151489258, 0.16226956248283386, 0.5402055382728577, -0.17342695593833923, -0.724185585975647], [-2.2079081535339355, -0.3807559609413147, 0.45007842779159546, -0.2910979688167572, -1.1855963468551636, 1.0085633993148804, -0.6799038648605347, 0.19623416662216187, -0.5091778039932251, -1.2655423879623413, -0.5519734025001526, 0.4783710539340973, 0.08504263311624527, 1.0267082452774048, -0.2865281403064728, 1.8212273120880127, -0.29754117131233215, 1.578844428062439, -0.2150535136461258, -0.00011100304982392117, -1.2012908458709717, 0.3845697343349457, -1.4454984664916992, 0.2608192265033722, 0.7695085406303406, 0.2892296016216278, 1.071548342704773, -1.0170572996139526, 0.10130129009485245, -0.4336565434932709, -0.5265335440635681, -0.4496026337146759, 0.6332242488861084, 1.8436373472213745, 0.3951650559902191, 0.16544099152088165, 2.368156671524048, 2.053928852081299, -0.7157748341560364, -0.7257778644561768, 1.6239759922027588, 0.13507691025733948, 0.22780752182006836, -1.1563202142715454, -0.5472569465637207, 1.2073659896850586, 0.2050878405570984, -0.9754159450531006, 0.9285914301872253, 0.32255738973617554, -0.41284167766571045, -1.513197898864746, 0.11193013191223145, 0.2540435194969177, 1.272853970527649, 1.888141393661499, -0.32374492287635803, -1.337602138519287, -0.2568269670009613, 1.1115719079971313, -0.497997909784317, -0.6059975028038025, -0.9992600083351135, -0.9472508430480957, 0.6952957510948181, 0.13691608607769012, 0.694299578666687, -1.0608958005905151, 0.7519000768661499, 1.794751524925232, -1.152277946472168, -1.2266243696212769, 1.2141258716583252, 0.12204642593860626, 0.45076918601989746, -1.4509340524673462, -0.6705639958381653, 0.6813525557518005, 1.1343539953231812, 0.22452642023563385, -0.14085005223751068, 0.8939555883407593, 1.6161683797836304, -0.028405901044607162, -0.199262335896492, 1.1094679832458496, 0.5325554609298706, 0.22156299650669098, 0.40452203154563904, -0.1741572767496109, -1.2522363662719727, 0.5625611543655396, -0.560184121131897, -1.2084566354751587, -1.1382285356521606, -0.5591832399368286, 0.012152977287769318, 1.372639536857605, 1.1358518600463867, -0.7943999767303467, 0.4414898157119751, -0.6109907031059265, -1.4904123544692993, 0.4871831238269806, -0.1051623746752739, -1.6665287017822266, -0.21605528891086578, -0.09282579272985458, 1.31141197681427, 1.8054507970809937, 0.618587076663971, 0.8571147322654724, 1.0748200416564941, -0.015678871423006058, -0.7231523394584656, 0.526064932346344, 1.4479831457138062, -0.7269131541252136, -0.5904385447502136, 0.8027592301368713, 0.5605431199073792, -0.3213704824447632, 0.8676798939704895, -1.2625797986984253, -0.40551021695137024, 0.44981449842453003, 1.4716053009033203, 0.45881780982017517], [-0.06677567213773727, -0.01744202710688114, -0.10723664611577988, -1.4979093074798584, 0.3827875554561615, 0.9282540082931519, -1.037846565246582, -0.18630437552928925, 0.16565293073654175, 0.010478384792804718, 1.7277624607086182, -0.5138366222381592, 0.6798896193504333, 0.5450999140739441, 0.48693227767944336, -0.8946651220321655, 0.68342125415802, 1.292662501335144, -0.9728311896324158, 0.4136578142642975, -0.028601637110114098, -0.7202025055885315, -0.04345148056745529, 1.0353477001190186, 0.9564947485923767, 0.9665533304214478, -1.2783186435699463, 0.5736688375473022, 0.11905200034379959, -0.17276866734027863, 1.284256935119629, -1.270235300064087, -1.0682367086410522, 0.9910141229629517, -0.21152810752391815, 0.8403179049491882, -0.8129348754882812, 0.2692539691925049, 0.8213557600975037, -1.088407039642334, 0.3517238199710846, 0.7304828763008118, -1.6593209505081177, -1.1996725797653198, 0.9839473366737366, 0.3921339213848114, -0.04730499908328056, -0.5946482419967651, 1.3281807899475098, 0.24861657619476318, -0.2978070080280304, -0.5816799998283386, 0.5018166303634644, -0.18144910037517548, 0.41592514514923096, -0.18009139597415924, 0.7273623943328857, -1.0786572694778442, 1.132278561592102, 1.1008448600769043, 0.4606543183326721, -0.0777672603726387, 0.25123822689056396, -0.4676339030265808, 0.14400143921375275, -0.9530362486839294, -1.316239595413208, -0.8386267423629761, 1.2865638732910156, 0.6568412184715271, 0.46271583437919617, -2.599560499191284, 0.47061049938201904, 1.492289423942566, 0.09583507478237152, -0.013147450052201748, 1.4844427108764648, -1.1096261739730835, -0.48849087953567505, -0.01407461054623127, -1.3559097051620483, 0.5347741842269897, -1.0516424179077148, 2.2654829025268555, 1.244686484336853, 1.1416304111480713, 0.6159576177597046, 0.24105921387672424, -1.0469797849655151, 1.4721542596817017, 0.3221563398838043, -0.27079781889915466, 0.5743092894554138, 0.5100784301757812, 0.9367366433143616, 0.3196581304073334, -0.6089293956756592, -1.404923915863037, -0.8193190097808838, -0.010836458764970303, -0.8734020590782166, 0.99949049949646, -0.01384648121893406, 1.1275640726089478, 0.2852388024330139, -0.3427668511867523, 0.15849906206130981, -0.7565264105796814, -0.14026454091072083, 2.3633644580841064, -0.03389724716544151, 0.4219358563423157, 0.6175246238708496, -0.6572434306144714, 0.1611565351486206, -0.6696351766586304, -1.935756802558899, 0.580730140209198, -1.9577192068099976, 0.28497567772865295, 1.2236930131912231, -1.2921221256256104, -0.4776718318462372, 1.0284701585769653, -0.7503494024276733, -0.7413192391395569, 1.2045224905014038, -0.9597535729408264], [0.6266219019889832, 1.0656695365905762, 0.36497774720191956, -0.19047366082668304, 0.5473300814628601, -0.29977190494537354, -0.6447175741195679, -1.061428427696228, 0.7140775918960571, -0.872035801410675, -0.6439953446388245, -0.17117267847061157, 1.0574275255203247, -0.999853789806366, -1.4910441637039185, -1.8526283502578735, -0.12482475489377975, -0.2489546537399292, 0.6813679933547974, -0.30586668848991394, -0.05576562136411667, -0.23617438971996307, 0.280491441488266, -0.6048162579536438, 0.09689734876155853, 1.231838345527649, 1.743669867515564, -0.5651826858520508, 1.2082020044326782, -1.96120023727417, 0.0237434022128582, -0.9421653747558594, -0.7540273070335388, 0.6483793258666992, 1.0407469272613525, -0.1616635024547577, 1.0563350915908813, -1.2836031913757324, -0.4817582666873932, 0.43739745020866394, -0.8292506337165833, 1.3917055130004883, 0.8807855248451233, 0.7413285374641418, 0.5096424221992493, 0.606616735458374, -0.07978800684213638, -1.3395318984985352, -0.6615533828735352, 0.3182814121246338, 0.05221723020076752, -1.4013330936431885, -0.17086005210876465, 0.7578516602516174, 2.588823080062866, -0.04764535278081894, 0.7411974668502808, 0.7335402965545654, 2.3779406547546387, -0.6090571880340576, 1.0289114713668823, -0.24920623004436493, -0.6296252608299255, 0.9507511258125305, -0.1968667358160019, -1.312722086906433, -0.3923794627189636, 0.7613840103149414, -1.2930505275726318, -0.8121621608734131, 0.23592136800289154, -0.061245180666446686, -2.4875640869140625, 0.0014181340811774135, -0.9676220417022705, -0.3112477660179138, 0.7339283227920532, -0.3455589711666107, -0.46440133452415466, -1.3368477821350098, -1.856539011001587, 1.86139976978302, 1.154090166091919, -0.5402824878692627, 0.14006048440933228, 0.4667230546474457, 0.22477012872695923, 0.15038612484931946, -0.23805056512355804, -0.9614651799201965, 0.6156486868858337, 0.8405546545982361, 1.4407901763916016, 0.22079670429229736, -1.9252779483795166, -0.2424904853105545, 0.7413623332977295, 2.169820785522461, -1.4376018047332764, -0.22117093205451965, -0.6816776990890503, 1.4880560636520386, -0.974273681640625, -0.07676678895950317, 1.187516450881958, -1.2291181087493896, -1.7949941158294678, 0.3095238208770752, 0.48759353160858154, 0.5828558802604675, -1.9878894090652466, 0.5006663799285889, -0.7798920273780823, -1.071959376335144, -0.9094926714897156, 0.24968698620796204, -1.3064703941345215, -0.15667055547237396, -0.7228443026542664, -0.16961412131786346, -0.7278364300727844, 1.5813015699386597, -1.0964865684509277, -0.16138611733913422, -0.2057550698518753, 0.7911387085914612, 0.21809303760528564, -0.9492507576942444], [1.4175885915756226, -0.10484596341848373, 0.19625942409038544, -0.33325889706611633, -0.20199455320835114, 1.2700486183166504, -0.2302960306406021, -0.7044429183006287, 0.0003902561729773879, -0.487418532371521, 2.337419271469116, -0.06674468517303467, 0.40940186381340027, 0.9587745070457458, -0.9218145608901978, 0.9930933713912964, -0.004019904416054487, 1.630460500717163, 0.13218095898628235, 0.8961275219917297, -0.7224218845367432, 0.22675810754299164, -0.4762032926082611, -1.5166385173797607, 0.21736416220664978, -0.27356237173080444, 1.4601889848709106, -0.030164610594511032, 0.796872079372406, 0.2627004086971283, 1.3601405620574951, 1.1292340755462646, 2.8239974975585938, -1.5608432292938232, -1.0628653764724731, 1.6187938451766968, -0.12573428452014923, -0.1903112679719925, -0.7846515774726868, -1.2856930494308472, -0.7259827256202698, -0.4463128447532654, 0.0597073957324028, 0.21043841540813446, -1.2883695363998413, 0.17208324372768402, 0.5086076259613037, -1.311959981918335, 1.4150232076644897, -0.7390036582946777, -0.18472807109355927, 0.37619277834892273, -0.8777397871017456, -2.1012909412384033, -0.5752623677253723, -1.0974596738815308, 1.022752285003662, -0.7961312532424927, -1.3715887069702148, 0.5123457908630371, 1.4500837326049805, -0.6195673942565918, 1.3468928337097168, -0.672116219997406, 1.6220747232437134, -1.1185925006866455, 0.6198204755783081, -0.180057555437088, -0.8875157237052917, -1.415116548538208, -0.6627956628799438, 0.02048593759536743, -0.7790586948394775, 1.1124902963638306, 0.5016106367111206, 0.1385670304298401, -0.3338978588581085, -0.6859645843505859, 0.18198269605636597, -0.7193092107772827, -1.3812849521636963, -1.6751655340194702, -1.0211750268936157, -1.0106004476547241, 0.027420897036790848, 0.4669235944747925, -2.2873528003692627, 1.762010097503662, 2.4762678146362305, 0.655407726764679, 1.2504020929336548, 0.1352405548095703, -1.477731466293335, 2.3952717781066895, -0.07899325340986252, 1.7003341913223267, 0.9609634876251221, 0.33139845728874207, -0.28764915466308594, 0.46147769689559937, -0.3948603570461273, -2.015636920928955, -0.6230033040046692, -1.3818084001541138, -0.10273273289203644, 0.4072955846786499, -0.21699611842632294, -0.11949636787176132, -0.1077282652258873, 2.0407609939575195, -0.355817049741745, -0.4325893819332123, -2.222378969192505, 0.8740999102592468, -2.175060272216797, 0.7587754130363464, 1.7424854040145874, -1.383730173110962, 0.23871468007564545, 1.3646622896194458, 1.6846954822540283, -0.3357376158237457, -0.2418106645345688, 0.42211005091667175, -0.8399908542633057, -0.6456827521324158, 0.10187357664108276, 0.050538454204797745], [-0.7061660289764404, 0.692658007144928, 2.1873090267181396, -1.0849425792694092, -1.513039231300354, -1.5439058542251587, 0.36168810725212097, 0.33640480041503906, -0.22683703899383545, 0.39830124378204346, -0.013602368533611298, -0.0478845089673996, 0.10994712263345718, 0.7870609164237976, 0.6240395307540894, 0.8136334419250488, 0.27176544070243835, -1.63878333568573, 0.25856658816337585, 1.476829171180725, 0.6700780391693115, 0.5001493096351624, -1.011460304260254, -0.5131648778915405, -1.1265138387680054, 1.1004128456115723, 1.6703522205352783, -0.4463617503643036, 1.681113362312317, -2.0044238567352295, -1.7460036277770996, -0.31328845024108887, -0.2905682325363159, 1.0970520973205566, 0.009378768503665924, -0.6050977110862732, 0.8395001888275146, -0.7775804996490479, 2.0737242698669434, 0.5698915123939514, 0.8025562167167664, -1.2021030187606812, -0.7716171145439148, 1.6355934143066406, -0.5884783267974854, -1.0923221111297607, 0.08019671589136124, 0.2722528874874115, 0.08738229423761368, -0.810801088809967, 0.1565750688314438, 0.8182085156440735, 1.447122573852539, 0.3423595726490021, 1.5011662244796753, -0.09632547944784164, -0.3768565058708191, 0.11807267367839813, -1.2003238201141357, 0.32224491238594055, -0.20910589396953583, 0.35243359208106995, 0.43897199630737305, 1.8521747589111328, -0.7321228981018066, 1.5620436668395996, -0.8922773599624634, -0.07105884701013565, -0.1846451759338379, 1.2013781070709229, -0.47683343291282654, -0.09483342617750168, 0.18848063051700592, 1.4654784202575684, -1.3433098793029785, -0.804851770401001, -0.5327048897743225, -0.24124674499034882, -0.018290361389517784, 1.5586930513381958, 1.0953763723373413, 0.13732893764972687, 1.0789469480514526, -0.7147956490516663, -1.6611645221710205, 0.7371875047683716, -0.7229716777801514, -2.1748740673065186, -1.8703550100326538, -0.08749954402446747, 0.6431605815887451, -0.48507896065711975, 0.9907203912734985, 0.5241549015045166, -2.7912817001342773, 1.08497154712677, -1.460076093673706, -0.04265300929546356, 0.3979432284832001, 0.019461292773485184, -1.465254545211792, -0.09925767779350281, -0.5670543313026428, -0.4193342924118042, 1.6943631172180176, -0.806833803653717, 1.610702395439148, 0.5759039521217346, -0.12046480178833008, -0.3763326108455658, -0.8313230872154236, 0.24612706899642944, -1.0959351062774658, 1.2344119548797607, -1.130405306816101, 0.8385875225067139, 2.3932387828826904, 0.8921683430671692, 1.1296570301055908, 0.6281738877296448, -0.4772859215736389, -2.566222906112671, 1.0292766094207764, -0.20150384306907654, -0.2446235567331314, -1.5588260889053345, 0.10837312042713165, 0.5245603322982788]], [[0.47125548124313354, -0.6189939379692078, 2.0814685821533203, -0.2427588552236557, -1.1514713764190674, -0.9611908793449402, 0.16262061893939972, -1.1822035312652588, 1.1746840476989746, 0.2205984890460968, -0.26792457699775696, -1.037065863609314, -2.897704601287842, -0.5774268507957458, -1.2739428281784058, 0.5290217399597168, 1.608672022819519, -1.8442389965057373, -0.005739486776292324, -0.08977053314447403, 0.14235633611679077, -0.8220767974853516, 0.5131890773773193, 0.7856281995773315, -0.6980470418930054, 1.5645359754562378, 0.4324565529823303, 0.2310071885585785, 0.37805262207984924, -0.9704540967941284, -1.8683823347091675, 1.5594046115875244, -0.10178915411233902, 0.01446966640651226, 0.565327525138855, 1.0188922882080078, 1.8734115362167358, -0.7584203481674194, -0.09532205760478973, -1.2948514223098755, 1.5012414455413818, -0.4294155240058899, -0.6991837620735168, -0.9005422592163086, 0.136734277009964, 0.0014120594132691622, -0.09737240523099899, 0.660575270652771, -0.43768396973609924, -0.2553723454475403, 2.390395402908325, 0.4753360450267792, 0.06234269216656685, 0.18686753511428833, 1.0193201303482056, -1.0564463138580322, -0.9968406558036804, -1.1613022089004517, -0.2921708822250366, 0.3422582149505615, 0.17658288776874542, 0.919873058795929, 0.26885971426963806, 0.4193045496940613, 0.9235241413116455, 0.7201040387153625, -0.08032403141260147, -0.39656737446784973, -1.4959526062011719, 1.5691167116165161, 1.2953495979309082, 1.5601764917373657, -1.652463674545288, 0.6598897576332092, -0.2462977021932602, -2.309499740600586, 0.08767368644475937, 0.44422242045402527, 0.8356568217277527, -0.9377472400665283, 0.1365160346031189, 0.7191482186317444, 0.9445116519927979, 0.8958982825279236, -0.25666818022727966, -0.36674943566322327, -0.25505703687667847, -0.44363799691200256, 2.1204302310943604, 0.7467591166496277, -1.7839068174362183, 0.2555396258831024, 0.035803839564323425, 0.3751453161239624, -0.3337458372116089, -0.5228429436683655, -0.760071873664856, -0.9075446128845215, 0.8121622800827026, 1.3310011625289917, -0.11322131007909775, -0.5646083354949951, -1.0093185901641846, 1.1846961975097656, 1.0677101612091064, 0.972001314163208, -0.9995871186256409, -0.04040847346186638, -0.3632190525531769, 1.5021133422851562, -1.4744564294815063, -0.5977907180786133, -0.7581935524940491, 0.704014778137207, 0.8817653656005859, -0.3638586699962616, 0.6995223164558411, 0.10926888883113861, 0.9105433225631714, -1.4953806400299072, -0.4170496165752411, -0.488487184047699, -1.1001758575439453, 2.180877447128296, -0.9430505633354187, -1.6948661804199219, 1.5096259117126465, -0.270683616399765], [1.2647461891174316, 0.7735980153083801, 1.7046412229537964, -0.7389764785766602, -0.8552024364471436, -0.4061911404132843, -0.4925871193408966, 0.7739692330360413, -0.1529235541820526, -1.9719934463500977, -0.7473045587539673, -0.6708125472068787, -0.9897660613059998, -1.2197108268737793, -0.3623439073562622, -0.28516700863838196, 0.6940402984619141, -1.6160356998443604, -0.02103033848106861, 0.002564165974035859, 0.26087918877601624, 0.04037575423717499, 1.391507625579834, 0.16351284086704254, 1.0926947593688965, -0.46824848651885986, 1.1150258779525757, -0.6103026270866394, -0.6180035471916199, -0.23726195096969604, -2.218458890914917, 1.155838131904602, -0.8425688147544861, -0.16307185590267181, 1.1791576147079468, -0.971139669418335, 0.2825625240802765, 1.5120611190795898, 0.5860462784767151, 0.7205662131309509, -0.09754869341850281, -0.45764797925949097, 0.15134398639202118, -2.6057662963867188, 0.33318665623664856, -0.613864541053772, 2.1351072788238525, -1.2650851011276245, 1.383609414100647, -0.12496765702962875, -0.5243707895278931, -1.30001699924469, -0.9686572551727295, 2.53934907913208, -0.7381786108016968, -1.2480638027191162, -0.6191854476928711, 0.9358009099960327, -0.66346675157547, -1.2963476181030273, 0.22084447741508484, -0.34347766637802124, 0.32612892985343933, -0.7722336649894714, -1.0597018003463745, 0.48593056201934814, -0.05877053365111351, -0.12100416421890259, -1.926038384437561, -1.3917640447616577, -0.31052273511886597, -0.057852499186992645, 0.7234990000724792, -1.9202760457992554, 0.06146099790930748, -0.19498705863952637, -1.0893803834915161, -0.6187226176261902, -1.4486628770828247, -0.12926802039146423, 0.16428157687187195, -0.319128155708313, 1.41122567653656, -0.8870476484298706, 1.1353118419647217, -1.335784673690796, 0.7885786294937134, -0.25032249093055725, 0.3742207884788513, 0.13485628366470337, -0.39404264092445374, -1.1890815496444702, -1.3605188131332397, 1.1148959398269653, 0.0003923453623428941, -0.1597810834646225, -0.2151404619216919, 0.5784671306610107, 0.2707521319389343, -0.35544243454933167, -0.13567101955413818, -0.21873213350772858, 1.2093226909637451, -0.21759313344955444, -1.4072209596633911, 2.099675416946411, 0.27490103244781494, -0.6455783843994141, -1.7902865409851074, 1.3338656425476074, -1.1506383419036865, -0.932589590549469, -0.7313740253448486, -0.011574812233448029, 1.1073890924453735, 0.773958146572113, -0.2041294276714325, -1.8600748777389526, -0.3643474578857422, -1.4386272430419922, -0.4453355371952057, -0.1415877491235733, 1.1103425025939941, -1.8244298696517944, -0.9054589867591858, 0.8943250179290771, 0.46324706077575684, 2.243196964263916], [0.6675958633422852, 0.6434422135353088, 0.7129621505737305, 0.624278724193573, -0.3659142851829529, -0.09875573217868805, -0.988084614276886, -0.41630008816719055, 0.004508500453084707, -0.14879584312438965, 1.2928928136825562, 0.7341824173927307, 0.45169955492019653, 0.3873879909515381, -0.6849180459976196, 1.8504676818847656, 1.252914309501648, -0.18688397109508514, 0.23310071229934692, -0.9239957332611084, 0.905327320098877, 2.0364019870758057, 0.7381004691123962, -1.6272695064544678, -1.4329330921173096, -1.1414930820465088, -0.21789155900478363, 1.694307804107666, 0.6385672688484192, -1.5997930765151978, 0.14019794762134552, 0.11434915661811829, -0.052610211074352264, 0.7634206414222717, 0.6515930891036987, 1.0381200313568115, -0.5985394716262817, -0.897225558757782, 0.23258079588413239, 0.2689535617828369, -0.24829138815402985, -0.21526722609996796, 0.9859890937805176, -0.764140248298645, 0.9452970623970032, -0.4438386857509613, -0.22412946820259094, 0.2528385519981384, 0.9848620891571045, -0.45803236961364746, -0.24806874990463257, -1.71905517578125, -0.11982957273721695, -0.4429904520511627, -1.5740246772766113, 0.31579121947288513, 1.8451542854309082, 0.4189421534538269, -1.1478118896484375, -0.6615909934043884, 0.9036681652069092, 0.7016634345054626, 0.9020918011665344, 0.2586890757083893, 0.114423468708992, -1.4460817575454712, 0.3636917769908905, 1.8639779090881348, 1.1111983060836792, -0.22701504826545715, -0.6893501877784729, 1.1083908081054688, 0.9684269428253174, 0.7831174731254578, 0.02568369358778, -0.5883110165596008, 0.5077464580535889, 0.07933726906776428, -1.0242469310760498, 0.13738815486431122, 1.0961757898330688, -0.8989874720573425, -0.03632038086652756, 1.6130746603012085, 0.18464531004428864, -0.3690185844898224, -1.28276789188385, -0.46715399622917175, 0.6235162615776062, 0.31002679467201233, 1.3776637315750122, -0.03552120551466942, 0.41929858922958374, -0.11123060435056686, 1.5175769329071045, -0.09311755746603012, 0.6413828134536743, 1.003013014793396, 2.462190866470337, 1.0085299015045166, -1.137082576751709, 0.29030394554138184, 0.03989189863204956, 1.0797319412231445, -0.5204418301582336, 0.051698196679353714, -0.23449239134788513, 0.2422703206539154, 2.465101718902588, 0.7401367425918579, 0.20470456779003143, 0.3639450669288635, 2.231090545654297, -0.0746520534157753, 1.7640999555587769, -1.682868242263794, 0.5162203311920166, 0.4477718770503998, -1.6188832521438599, -0.616919994354248, 1.0019465684890747, -1.0139882564544678, -0.23831282556056976, 0.014113848097622395, 0.11786109954118729, 0.2515692710876465, 1.4717572927474976, -1.5264171361923218], [-0.055716581642627716, -0.4749114215373993, -0.20273733139038086, 0.4094754755496979, 2.4369451999664307, -0.07333201169967651, -2.900669813156128, -0.10060327500104904, -0.5716846585273743, -0.27091994881629944, -0.17556963860988617, -0.1668553650379181, -0.4183824360370636, -3.060596227645874, 0.9082139134407043, -0.5293890833854675, -0.23187175393104553, -1.678072214126587, -0.10278062522411346, -0.5878168344497681, 0.7974352836608887, -0.08159667253494263, -1.5644607543945312, -2.129565477371216, 0.14495886862277985, 0.2844133973121643, -0.9372637271881104, 1.141406774520874, -0.45866748690605164, -0.3931798040866852, 2.292126178741455, 1.4236671924591064, 1.547296166419983, -0.7184459567070007, -0.9686576724052429, -0.4634368419647217, -0.6115188002586365, -0.9128065705299377, 1.0512899160385132, 0.37161484360694885, 0.3122378885746002, -0.4495386481285095, -0.7513462901115417, -0.5806728005409241, -2.1541991233825684, 0.03491036221385002, -1.235608696937561, 0.17177145183086395, -0.5076690912246704, -0.4373459219932556, 0.5285946130752563, -1.3310210704803467, -1.4828298091888428, 1.0662615299224854, 0.5155418515205383, 0.42878246307373047, -0.6281706690788269, -0.43839383125305176, -1.016052484512329, -0.7667568325996399, -0.4425341784954071, -1.0168918371200562, -0.7451484799385071, 0.3976893723011017, -0.7179211378097534, -0.631462812423706, 0.2137852907180786, -0.28800034523010254, 0.5024836659431458, 0.5398995280265808, -0.20457951724529266, -0.2817997932434082, 1.1209797859191895, 0.00019252393394708633, 0.4449605643749237, 0.4937931001186371, -0.7059947848320007, -0.22395028173923492, 0.13787683844566345, 0.23649106919765472, -0.25315096974372864, 0.3535233736038208, 0.855958104133606, 0.8126344084739685, -0.12591461837291718, 0.5964646935462952, 0.02521231211721897, -1.2611616849899292, -0.22382959723472595, -1.2769461870193481, 0.2804637551307678, -0.6126474738121033, 0.14053106307983398, -2.903749465942383, -2.409885883331299, 0.7545121312141418, 0.23075778782367706, 1.2100379467010498, -1.4957741498947144, 0.656723141670227, -0.7986869215965271, 1.7499964237213135, 1.4631589651107788, 0.037104759365320206, -0.5868706107139587, -0.591196596622467, 0.10313054174184799, 1.0725442171096802, -0.8826273679733276, -1.3136112689971924, -0.40282702445983887, 0.030576875433325768, -1.6237068176269531, -0.619434654712677, -0.7124428749084473, 0.9474385380744934, 0.46550580859184265, 0.28023505210876465, -0.7786675691604614, -0.1327488273382187, -0.568129301071167, 0.8720224499702454, -0.8600283265113831, 0.046773336827754974, 0.28938350081443787, -0.1195683628320694, -1.7123769521713257, 1.4956218004226685], [-0.3693617582321167, -0.46597030758857727, -0.11426199972629547, 0.7758885622024536, -0.20513610541820526, 1.1022616624832153, 2.2727203369140625, -2.002286672592163, -2.2787880897521973, -0.11901319026947021, -0.6164646148681641, 0.2521354556083679, 1.3765928745269775, -0.6600401997566223, 1.1158825159072876, 0.05554789677262306, -1.1667903661727905, 0.702239990234375, 0.7679377794265747, -0.24836421012878418, 1.0340867042541504, 0.5062678456306458, 0.9335306882858276, -1.8008309602737427, -0.22047767043113708, -0.6874139904975891, 1.6063635349273682, -0.30877408385276794, -0.37678512930870056, 1.3137798309326172, 0.037038810551166534, -0.20940355956554413, -1.9579405784606934, 1.5018413066864014, 1.278969407081604, -0.5483583807945251, 1.555220603942871, -1.1379212141036987, 0.288718044757843, -1.3872482776641846, 0.28449156880378723, -0.17510727047920227, 1.603703260421753, -0.9038145542144775, -0.5030839443206787, 0.26812854409217834, 1.0982065200805664, -2.102910280227661, 0.07161147892475128, -1.8401206731796265, -1.4504897594451904, 0.8918229341506958, -1.4091767072677612, -0.5404724478721619, -2.3595621585845947, 0.9325235486030579, 0.7954818606376648, 1.6934486627578735, 0.6984764337539673, 0.6158711314201355, -0.43631505966186523, -0.6259152889251709, -1.1101760864257812, -0.6086127161979675, -0.6465911865234375, 0.31411027908325195, -1.2072795629501343, -0.10190454125404358, 0.3183797001838684, -0.5993025898933411, 0.026102744042873383, 1.3733737468719482, 0.7193411588668823, -0.23497071862220764, -0.3451276421546936, -1.191885232925415, 0.6392600536346436, 1.0715000629425049, 1.47970712184906, -1.6333234310150146, -1.4173510074615479, -0.2012729048728943, -0.6879163384437561, -0.7085995674133301, 0.6785327792167664, -0.29533520340919495, 0.6372120976448059, 1.2204066514968872, -0.19574017822742462, 0.40478503704071045, -0.006760458927601576, -0.6710575819015503, -1.477565050125122, -0.40143200755119324, -0.31026244163513184, -0.37268051505088806, -1.4174176454544067, 0.5903137922286987, 1.1381672620773315, -2.842297077178955, -0.8039877414703369, -0.11007653176784515, -0.3855789601802826, -2.181661605834961, 0.3956278860569, 0.5857276320457458, -1.7344696521759033, 0.9487010836601257, -0.06851475685834885, 1.1851253509521484, 1.1991865634918213, -0.0943945050239563, 0.16276662051677704, -0.5595384240150452, 0.26578736305236816, 1.48464834690094, -1.0383838415145874, -0.29041534662246704, -0.3016277849674225, 0.9759076833724976, -0.0015005021123215556, -0.5400540232658386, 0.35233673453330994, 0.835934579372406, 0.10883768647909164, -0.7081891894340515, 0.9482529759407043, -0.6729205250740051], [-1.233659267425537, -0.1354122906923294, -0.25120168924331665, 1.5830172300338745, 0.12286804616451263, -1.3996050357818604, 0.2794136106967926, -0.17604507505893707, 1.0417916774749756, -1.277873158454895, -0.5008133053779602, -1.4141321182250977, -1.8021305799484253, -1.0684740543365479, -1.7465237379074097, -0.5779337882995605, 1.7730867862701416, -0.34858521819114685, -1.59246826171875, -0.13336731493473053, 0.3032667636871338, -0.17246410250663757, 0.28801795840263367, 1.8374028205871582, -0.6468340754508972, -0.846642017364502, -0.5857062339782715, -0.35752731561660767, -0.9020398259162903, 0.6451420187950134, 0.9954351186752319, -1.036617398262024, -0.32136887311935425, -0.9428009986877441, -0.43383902311325073, -0.5467720627784729, -0.7539215683937073, 2.523474931716919, -1.3531800508499146, 1.039101004600525, 1.6361943483352661, -2.4808928966522217, 1.8987339735031128, 1.51237952709198, 0.7710910439491272, 1.0945074558258057, -1.03848135471344, 0.6499107480049133, 0.6679115295410156, -1.523846983909607, 0.8293505907058716, -1.499992847442627, 0.7323863506317139, -0.4907877445220947, 1.5137100219726562, -0.6330452561378479, -0.5122068524360657, -0.1297670155763626, 0.22524571418762207, 0.40412116050720215, 0.9554161429405212, 0.5325267910957336, 0.4438616633415222, 1.0498708486557007, -1.1934922933578491, 0.5624244809150696, 1.2153924703598022, 2.1321632862091064, -0.24178634583950043, 0.8459519743919373, -0.7201826572418213, 1.0767358541488647, 1.304887056350708, 1.6768572330474854, 0.4774029552936554, -1.1327028274536133, 2.01884388923645, -2.218315839767456, 0.7896257042884827, -0.2675558924674988, 0.3791607618331909, 0.3774474859237671, -1.741716980934143, 0.21941153705120087, 1.3254783153533936, -1.2843036651611328, 1.2046104669570923, -0.8683457970619202, 0.5796394348144531, -1.2787978649139404, 0.2033480852842331, -0.7847191095352173, -0.8410245776176453, -0.9454007744789124, -0.8700456023216248, -0.8471996188163757, 1.0380620956420898, 1.9889980554580688, 0.5242739915847778, 1.4456815719604492, -0.5356647372245789, 0.03335629031062126, 0.38641712069511414, -1.8528579473495483, 1.675154447555542, -0.6449437141418457, 0.5744662880897522, 2.304835319519043, 0.38177207112312317, -0.37846362590789795, -2.2180662155151367, 1.5040379762649536, -0.45790085196495056, -1.483780026435852, 2.031986713409424, 0.24655739963054657, -0.6568881869316101, -0.4513814151287079, -0.13346117734909058, 1.1569385528564453, -0.3743320107460022, 0.9336990714073181, -0.2945297062397003, -1.3562383651733398, 0.2740364968776703, 0.6903754472732544, -1.1842600107192993, -0.6734015345573425], [-1.6705681085586548, 1.0597312450408936, -0.3928287923336029, -0.24762003123760223, -0.4332501292228699, 0.7031866312026978, -1.0647773742675781, -0.8824635744094849, -1.0064315795898438, -0.541918158531189, 1.396437168121338, -1.0079641342163086, 1.2617214918136597, 1.0875343084335327, -0.0897863358259201, -1.8144890069961548, -0.5491622686386108, -1.1975817680358887, 0.6063149571418762, 0.11360806226730347, 1.1007484197616577, 0.2376052439212799, 0.7105314135551453, 0.6793480515480042, 1.2513103485107422, -1.7469772100448608, 1.0558077096939087, -0.268811970949173, 0.7585999369621277, -1.7489358186721802, 0.18048939108848572, 0.9387888312339783, 0.28254520893096924, -1.3031022548675537, -0.40693438053131104, -0.38755884766578674, 0.31170526146888733, 0.41684582829475403, 0.04274671524763107, -0.009008887223899364, -1.5118173360824585, 0.32592853903770447, -1.2878813743591309, 0.6564143896102905, 1.174456000328064, -0.48699307441711426, -0.7237759232521057, 1.2200706005096436, -0.5323737859725952, 0.475952684879303, -0.38344135880470276, 0.9172493815422058, 0.3682423532009125, -0.9147320985794067, 0.22676001489162445, 0.7075349688529968, 1.7517949342727661, -0.8158872127532959, 0.1768244355916977, -0.14104381203651428, 0.2616763114929199, 0.10631804913282394, 0.3235204815864563, 0.3593331575393677, -0.38324910402297974, -0.07134567201137543, 1.4858578443527222, -0.7715228199958801, -1.5004069805145264, -0.5874694585800171, -0.22156061232089996, -0.25887688994407654, -0.2988668382167816, 1.1355448961257935, 2.673191785812378, -0.5750311613082886, 1.0889049768447876, -0.605689287185669, -0.3849908113479614, 0.2788323163986206, 0.725056529045105, 0.8506705164909363, 0.9977239966392517, 0.9781882166862488, -0.09515809267759323, 0.8992317318916321, 0.6962037682533264, -1.7198725938796997, -1.3754173517227173, -1.3011293411254883, 0.5118709802627563, 1.3605635166168213, -1.0325700044631958, -0.5149564743041992, -0.08949978649616241, 0.06755596399307251, 0.8249276876449585, -1.8591806888580322, 0.8398029804229736, 1.1585593223571777, 0.8476547598838806, 0.4730587303638458, -0.19305337965488434, -0.5919705629348755, -0.031825434416532516, 0.1911982148885727, 0.6745110154151917, 2.634394645690918, 0.7146672010421753, 0.7731522917747498, -0.9475187063217163, 0.04589022696018219, 0.7422530651092529, -0.6698926687240601, -0.8708621263504028, -2.3987953662872314, 0.8582189083099365, 0.5922999978065491, -0.6366371512413025, -0.8412267565727234, 0.0957416519522667, 0.9499816298484802, -1.5735591650009155, 0.02662605047225952, -0.3453536927700043, -0.8475016951560974, 0.8257227540016174, -0.39778244495391846], [0.5007467865943909, 0.9216036200523376, 0.22322531044483185, -0.5580227375030518, -0.5342227816581726, -1.2247792482376099, -0.951124370098114, -1.2548075914382935, 1.3031378984451294, -0.029295312240719795, 0.09096530824899673, 1.2667045593261719, -0.2820740342140198, 0.4889606833457947, -1.5123074054718018, 1.159019112586975, 0.1041155532002449, -0.11293797194957733, -0.2118954360485077, -0.7253766655921936, 0.15194040536880493, -0.35410043597221375, 0.9745801687240601, -0.18993665277957916, -0.7025418877601624, -0.29229193925857544, -1.4049854278564453, 0.05098703131079674, -0.6662617325782776, 0.9517125487327576, -0.14735101163387299, 0.10572104156017303, -0.7136306762695312, 2.333402633666992, 0.7980008721351624, -0.5291407108306885, -1.8357765674591064, 0.7942498326301575, 0.12801824510097504, -0.628196656703949, 0.8917311429977417, 0.22642002999782562, 0.6813337206840515, 0.556938648223877, 0.5323932766914368, 0.17441724240779877, 0.38336512446403503, -0.21799327433109283, -0.6247856020927429, -0.8342503905296326, -1.1691391468048096, -1.9752434492111206, -0.6580719351768494, -1.2465357780456543, 0.6603354215621948, 1.6185224056243896, -1.2286438941955566, -0.43698862195014954, 1.6955658197402954, 1.2312681674957275, 1.4080395698547363, -1.8355852365493774, -0.860000729560852, -0.752600908279419, 0.46962690353393555, -1.7672595977783203, 1.3155899047851562, -1.5279303789138794, 0.5219779014587402, 1.3312970399856567, -0.724678635597229, 0.8279003500938416, -1.1442803144454956, 0.9183310270309448, 1.9675043821334839, -1.1056934595108032, 0.256352037191391, -0.46541351079940796, 0.7927452325820923, 2.187443256378174, 0.2534622848033905, -0.434234082698822, -0.1223779246211052, -0.3032985329627991, -0.3534841239452362, -0.805182695388794, -0.9335014820098877, 0.20500217378139496, -0.5695949196815491, -0.86721271276474, 1.0391989946365356, 0.6211327910423279, -1.7482635974884033, 1.3487809896469116, -0.20864032208919525, 1.366994857788086, -1.588141918182373, 0.8739800453186035, 0.2359902411699295, -0.9708374738693237, 0.5710773468017578, -0.007881777361035347, -0.15459449589252472, 1.110107421875, 1.8889439105987549, 1.6591566801071167, -0.7613041996955872, 1.1501020193099976, -0.21923820674419403, 0.23415130376815796, 1.4275858402252197, -1.2136493921279907, 0.08652927726507187, 1.3709100484848022, -1.229635238647461, -0.6163012981414795, 0.5012023448944092, 0.4919638931751251, 0.4921776056289673, 0.3272864818572998, -0.4955991804599762, -0.2903292775154114, 0.8823554515838623, -0.1254684031009674, 0.7784620523452759, 2.1741652488708496, -0.1802079975605011, -0.1047133058309555]]], [[[0.21344543993473053, -0.49211177229881287, 0.6335144639015198, -1.8557064533233643, 0.16207030415534973, 0.8550379872322083, -0.4869137108325958, -0.8671894669532776, 0.33400866389274597, -0.46367591619491577, 0.9125679731369019, -0.5860824584960938, -2.2520601749420166, 1.2806118726730347, -0.7741931676864624, -1.1679250001907349, 1.1805377006530762, -0.393957257270813, -0.09653020650148392, -0.9913179874420166, 0.12752948701381683, 1.272359848022461, 0.010250350460410118, -0.4918193221092224, 0.354900598526001, 0.5693711042404175, 0.589542806148529, 0.33654695749282837, 0.386165052652359, -0.44639289379119873, -0.3591490089893341, 0.9135409593582153, -0.43904489278793335, -0.4611247777938843, 2.4415957927703857, -2.33028244972229, -0.9020301103591919, 0.5115411281585693, -0.5444105863571167, 1.436766505241394, -1.4424065351486206, 1.1192903518676758, 0.3825104534626007, -1.5552842617034912, -1.4087046384811401, -0.38895857334136963, -2.436840057373047, -0.40806111693382263, -1.716662883758545, -0.6013790965080261, -0.10829780250787735, -0.007535883225500584, 0.7685093283653259, 0.8214373588562012, 0.06418630480766296, -1.610032558441162, -0.6647781133651733, 0.4975318908691406, -0.8453249931335449, -1.7703320980072021, 1.7428946495056152, 1.136434555053711, 1.3538479804992676, -0.14642371237277985, -0.07694466412067413, 1.7709381580352783, 0.21496552228927612, 0.9746822714805603, 0.9079504609107971, 0.2651413083076477, -0.5099958777427673, -0.2869056463241577, -0.36641138792037964, 0.7782705426216125, 0.9435924887657166, -2.5024802684783936, 0.34916266798973083, 0.9879421591758728, -0.8455316424369812, 1.3763693571090698, -1.166626214981079, 1.1639714241027832, 0.48164859414100647, -0.07214102149009705, -0.506977379322052, -0.29699626564979553, -0.4981561601161957, 1.7730282545089722, 1.5255547761917114, 0.5397740006446838, 0.7835322022438049, 1.0646523237228394, 0.27247509360313416, -0.7745960354804993, -0.19278180599212646, 1.059829592704773, 0.2026078999042511, -0.6296629905700684, 0.1907017081975937, 0.43776172399520874, -0.4080609977245331, 2.5876853466033936, 0.5943648219108582, -1.9728431701660156, 0.3609021008014679, -0.7538818120956421, -0.18157707154750824, 0.2555141746997833, 0.9597761631011963, -0.6579241752624512, 0.21569640934467316, -0.16613611578941345, -0.3796197175979614, 1.080039620399475, -0.0682513564825058, -1.541533350944519, -1.4997137784957886, -0.2529263496398926, -0.22673237323760986, 0.5035735964775085, 0.6436243653297424, 0.06638623028993607, -1.3619771003723145, -0.27226728200912476, -0.9041834473609924, 0.39937448501586914, -0.8438131213188171, -0.19621942937374115], [-0.762913703918457, 2.433061361312866, 0.17800289392471313, 0.46162956953048706, 0.722044050693512, -1.3612020015716553, -0.3151698708534241, 1.5365086793899536, -0.18071149289608002, 0.02399839274585247, -0.7288482785224915, 0.812875509262085, -0.78110671043396, -1.3733820915222168, 0.8047616481781006, -0.616592526435852, -0.09234549850225449, 1.0119127035140991, -0.9925715923309326, -1.060295581817627, -0.13359801471233368, -1.370376467704773, 0.879630446434021, 0.38150089979171753, -0.5585959553718567, -0.35461822152137756, -1.1281135082244873, 0.09009704738855362, 0.2647920548915863, -0.754040002822876, -1.1307857036590576, 0.41044068336486816, -1.1685051918029785, 0.3213596045970917, 0.37881794571876526, 0.7796860337257385, -1.2330378293991089, 0.37253671884536743, -1.2646299600601196, -0.3470182716846466, -0.3230239450931549, -0.7715299129486084, -0.7845309972763062, 0.830390214920044, -1.7276867628097534, 1.5253627300262451, 0.4557819664478302, 0.06195218116044998, -2.8428354263305664, 0.020313745364546776, -0.9940013289451599, -0.7932396531105042, 0.9603474140167236, -0.47567126154899597, -0.4078527092933655, 0.39330974221229553, -1.0249557495117188, -0.6888086795806885, -1.3745850324630737, 1.7667005062103271, 2.48405122756958, -1.19524085521698, 0.05207596719264984, -0.9644792675971985, -0.7553430199623108, 0.3729783594608307, -0.6710032224655151, 0.889775812625885, -1.719411849975586, 1.8444432020187378, 1.0436151027679443, 0.018787285313010216, -0.33988842368125916, -1.0768647193908691, 0.46596646308898926, -0.32741621136665344, 0.49247562885284424, 0.4725164473056793, -0.520545482635498, -1.1314655542373657, -1.3809081315994263, -0.5189192891120911, -1.9685566425323486, -0.21236646175384521, -0.9299747347831726, 0.728118896484375, -1.234112024307251, 0.08939728885889053, -0.7444292902946472, -0.9478126168251038, 1.7703684568405151, 0.06917912513017654, -0.11800269782543182, 0.046414636075496674, -2.0811500549316406, 0.8942763805389404, -0.07817277312278748, 1.3934911489486694, 2.041721820831299, -0.5655545592308044, 2.0656962394714355, -1.3376824855804443, -1.045270323753357, -0.16772282123565674, 0.14876237511634827, -0.0470268614590168, -0.6348888278007507, 0.470316618680954, -1.4055017232894897, -0.9958858489990234, -0.5323649048805237, 1.0834496021270752, 0.4376561641693115, -0.4126680791378021, 2.2134904861450195, -0.35388049483299255, 0.7019833922386169, 0.8059570789337158, -1.8056637048721313, 2.0405595302581787, 1.5777438879013062, -1.2250686883926392, 1.4868721961975098, 0.9804887175559998, 0.4948561191558838, -0.8227676153182983, -0.7519480586051941, 0.31780585646629333], [0.7810820937156677, -1.2410439252853394, -1.5102925300598145, 0.19840866327285767, 0.5692209601402283, -0.6847190856933594, 0.2403387576341629, -1.3229308128356934, -0.9179531335830688, -1.134426236152649, 1.7433744668960571, -0.6588704586029053, -0.37798017263412476, 0.13091248273849487, 0.1541386991739273, 0.5260553359985352, 0.8917555809020996, -0.36729443073272705, -0.2336275428533554, -0.10830363631248474, 0.7546058893203735, -0.690161406993866, 1.8877341747283936, -0.2273956835269928, -0.2753303050994873, -1.8660038709640503, -0.8619707226753235, -0.5702549815177917, 0.5885300040245056, -0.4261312186717987, 2.4871721267700195, -0.36906731128692627, 0.8204531073570251, -1.2779076099395752, -0.017933949828147888, 0.1996428370475769, -0.6807484030723572, 0.43846601247787476, 0.6271030306816101, -0.32009652256965637, -1.134860873222351, -1.2111133337020874, 0.9036523699760437, 0.10663734376430511, 0.3833441734313965, -1.469897985458374, -1.8236677646636963, -0.6909236311912537, -0.1294577568769455, -1.0327686071395874, -1.0203129053115845, -0.03393439203500748, -0.9652939438819885, -0.03420160338282585, -0.4931151270866394, 0.3783940374851227, -1.099614143371582, 1.6975940465927124, 0.4857456386089325, 1.0022609233856201, 0.4905641973018646, -2.1441564559936523, -0.08612719923257828, -0.4619148373603821, -0.23634815216064453, 0.21285560727119446, -0.7672753930091858, 0.10741448402404785, -0.6975666284561157, -0.060180407017469406, 0.2861591577529907, 0.3624292016029358, -0.04609519988298416, 1.9443238973617554, 0.9589350819587708, 1.271386742591858, 0.40733423829078674, -2.0341358184814453, 1.6413356065750122, -0.05452103912830353, 1.1465504169464111, 0.4161701202392578, 1.5666345357894897, -1.0401984453201294, 0.2708822786808014, 0.05971657857298851, 1.0458660125732422, 1.307065725326538, 0.08117056638002396, 0.3962325155735016, 0.9633915424346924, 0.5646577477455139, -0.011553332209587097, -1.320554494857788, 0.9217587113380432, -0.8346985578536987, 0.38037651777267456, 0.7353373169898987, -0.213909313082695, 0.6314454674720764, -1.0697826147079468, 0.4158903658390045, 0.8065072894096375, -1.3987839221954346, 0.04837329685688019, -0.7196446061134338, -0.8858428001403809, -0.6636173129081726, 0.10541458427906036, 0.9817190170288086, 2.3574295043945312, 0.795529842376709, -0.7975704669952393, -0.3508766293525696, -0.5401731133460999, -0.13269416987895966, 0.43809774518013, 1.1884647607803345, 0.982755720615387, -0.7221661806106567, 0.2649257183074951, -0.36192113161087036, 1.0002871751785278, -1.4185727834701538, -0.7689863443374634, 1.0263440608978271, -0.8665812611579895, 0.36249953508377075], [-0.3940330445766449, 0.27108529210090637, 0.5450748801231384, -0.7653968334197998, 1.4865329265594482, -0.6025406718254089, 0.7454151511192322, 1.342653512954712, 0.4044193625450134, -0.41291341185569763, -0.7704535126686096, 0.14782634377479553, -0.6547902226448059, -1.5314087867736816, -0.45420730113983154, -0.4829866886138916, 0.10278815776109695, -0.2911914587020874, 1.5196501016616821, -0.5462928414344788, 0.8826791644096375, -1.5563257932662964, -1.5813117027282715, -0.21565833687782288, 1.013563871383667, 0.6995795965194702, -1.2804335355758667, 3.1605842113494873, -0.7002931237220764, -0.593912661075592, 0.34467563033103943, -1.8128859996795654, -0.23423069715499878, -1.705671787261963, -0.23964639008045197, -1.8231539726257324, -1.4242632389068604, 0.49569815397262573, 0.5871658325195312, 0.631659984588623, -0.5487908124923706, -0.2749413251876831, -0.6478532552719116, -0.6021860241889954, -0.8284935355186462, 0.0409311018884182, -0.408908873796463, -0.6093286871910095, -0.2088998258113861, -0.7536783814430237, 0.32207441329956055, -1.371214509010315, 0.5386488437652588, 0.21736590564250946, -0.39990583062171936, -1.778836727142334, -0.6789687871932983, -0.14119715988636017, -0.19592706859111786, -2.011474370956421, -1.438939094543457, -0.6172282099723816, -0.2037373036146164, -0.637346088886261, 1.1614280939102173, -1.0111945867538452, -2.3373560905456543, -0.4467787444591522, 1.2424980401992798, 2.0092694759368896, -0.45099174976348877, -0.9375317096710205, -0.39134329557418823, -1.2346553802490234, -0.04395626485347748, -1.6089929342269897, 0.5124753713607788, 0.24071353673934937, -0.08250148594379425, -0.9863521456718445, -1.4388281106948853, 0.34639954566955566, 1.487364411354065, 1.9713503122329712, -1.046692967414856, 0.5072612762451172, 0.9455006718635559, 1.3754316568374634, 0.016682937741279602, -0.9727343320846558, 0.20658664405345917, 1.8018969297409058, -0.1690698266029358, 0.8621377944946289, -0.36315011978149414, 2.4331655502319336, 0.2961371839046478, -1.4339208602905273, -0.36382225155830383, 1.3938872814178467, -0.02478072978556156, -0.1713947355747223, 0.39764925837516785, -0.5365879535675049, 0.4950653612613678, -0.7034848928451538, 0.9929757714271545, 0.727816641330719, -0.2351597547531128, 1.2302827835083008, -0.490774542093277, -0.02484486624598503, -2.174983024597168, -1.4391669034957886, -0.06990768760442734, 1.2051821947097778, -0.8615769147872925, -0.6858619451522827, -0.6423400044441223, 1.3784235715866089, -0.4173488914966583, 0.5954895615577698, -0.07803013175725937, -1.033268928527832, 1.0691864490509033, -1.3434323072433472, -0.5442119240760803, -1.44990873336792], [-0.12550562620162964, 0.824407696723938, 0.6232852935791016, -1.293001413345337, 1.9253735542297363, 1.576488971710205, -0.5927104353904724, -0.01348799653351307, 0.02124762535095215, 1.357673168182373, 1.2116039991378784, -0.40165773034095764, 0.9490537047386169, 0.20647403597831726, -1.862677812576294, -1.1352397203445435, 0.4760999083518982, -0.20310385525226593, 0.6573805809020996, -0.5814955234527588, -0.5713973641395569, -0.9215046167373657, -1.1234705448150635, 0.10054556280374527, 0.8591082096099854, -0.3047860562801361, 0.15961787104606628, -0.24228383600711823, 0.17185139656066895, -1.5488277673721313, 0.9721858501434326, -0.8733809590339661, -0.3204267919063568, -0.815991222858429, -1.8626168966293335, 1.0641162395477295, 2.0861828327178955, -1.456270694732666, 1.2069454193115234, -1.0930171012878418, -1.305781602859497, -2.122091054916382, 0.7572465538978577, -1.1759467124938965, -0.5596474409103394, 0.887330174446106, 0.49118876457214355, 1.3701658248901367, 3.0175232887268066, -1.7373175621032715, -0.9926794171333313, 0.7183748483657837, -0.4009222388267517, 1.069179892539978, -0.5370411276817322, -1.4854332208633423, 0.9985514283180237, -0.6382997632026672, 1.7883185148239136, 0.95782870054245, -0.11599108576774597, -0.3312135934829712, 0.4399425983428955, -2.122002124786377, -0.06094982847571373, 0.6121554970741272, -0.50218266248703, -0.191621795296669, -0.05481914430856705, 0.7789705395698547, -0.4525114893913269, 0.34378984570503235, 0.4071781039237976, 1.2994251251220703, -0.5617650151252747, 0.41126707196235657, 0.4274110198020935, -1.0390411615371704, 2.354135274887085, -3.5219836235046387, -0.27354517579078674, -0.28708377480506897, -1.0754282474517822, -2.1705949306488037, -0.447051465511322, -1.1147209405899048, 0.4322066009044647, -0.7716111540794373, -0.4538106620311737, -0.4399680495262146, 0.09290647506713867, 0.29573479294776917, -1.5549216270446777, -0.184552863240242, -0.5431607961654663, -0.15610778331756592, 0.911483645439148, 0.8907297849655151, 0.38306382298469543, -0.4022919237613678, 0.07262211292982101, -1.6127992868423462, 0.46992918848991394, -0.5476841926574707, 0.9783614873886108, 0.2355899065732956, 0.033674366772174835, 2.2080094814300537, 1.019826054573059, -0.947381317615509, -1.0639443397521973, -0.25561314821243286, 1.803406000137329, 0.038702260702848434, 2.4994258880615234, 1.0876396894454956, 0.3856733739376068, -0.5119537115097046, 1.4292645454406738, 0.9127357602119446, 0.17945201694965363, 0.21104896068572998, -0.5404210090637207, 0.32564541697502136, 1.4050960540771484, -0.3530493676662445, -0.3354184329509735, -1.9241511821746826], [-0.706093430519104, 1.2550404071807861, 0.3062547743320465, -0.11224211752414703, -0.15663069486618042, -0.9208134412765503, 0.027426807209849358, 1.5539484024047852, -0.42955973744392395, -1.9108433723449707, 0.30510467290878296, -0.18325282633304596, -0.5657311081886292, 1.6401768922805786, 1.0876764059066772, 1.2716548442840576, 0.31519249081611633, -0.5690072774887085, 0.23161402344703674, 0.4098149240016937, -0.10992449522018433, 1.5773096084594727, -0.7878497242927551, 1.403977870941162, -1.0542237758636475, -0.09424196183681488, -0.879902184009552, 0.08107833564281464, 0.6627103090286255, 0.17059749364852905, 0.38838014006614685, 0.7423406839370728, -0.32965466380119324, -1.5527818202972412, -2.0923709869384766, -0.5898964405059814, 0.4611615538597107, 1.356177806854248, 1.532060980796814, 1.3984910249710083, -0.9149269461631775, -1.280517578125, 0.6604238748550415, 0.4887317717075348, -1.3349740505218506, -0.6540929675102234, 1.5570589303970337, 0.3683772087097168, -0.05262456461787224, 1.8083899021148682, 0.9620932340621948, 0.8181843757629395, -0.44414716958999634, 0.04635591432452202, 0.626778244972229, 1.440873146057129, 2.1701064109802246, 0.6182786226272583, 0.16829949617385864, -1.677838921546936, -0.5072048306465149, 2.1079630851745605, 0.2503341734409332, 0.6389470100402832, -0.2984638810157776, -0.6027877330780029, 0.6643354892730713, 2.341399908065796, -0.4313699007034302, -0.7810139060020447, 0.3199141323566437, 1.184971809387207, -1.2096761465072632, -2.7447686195373535, 0.36196368932724, 0.5180068612098694, -1.1644704341888428, 0.8824573755264282, 0.765149712562561, 1.257925271987915, -0.143232062458992, 0.06001749634742737, -0.2970748543739319, -0.3119240999221802, -0.2798691391944885, -1.9572365283966064, 1.1823030710220337, -0.6555652022361755, -2.0649001598358154, 1.6585417985916138, 0.8264063596725464, 1.0924968719482422, 0.053400974720716476, 1.008695125579834, -1.1818304061889648, -0.3528427183628082, 0.9811991453170776, 0.47642311453819275, 0.380883127450943, -0.641837477684021, 1.067899227142334, 1.3074103593826294, -1.112310528755188, 0.74159175157547, -1.5983396768569946, 1.6161038875579834, -0.15749818086624146, 1.4457347393035889, 1.9494389295578003, -0.034164898097515106, 0.14052550494670868, -0.06720951199531555, 0.580222487449646, -0.31124863028526306, -0.5552718639373779, 2.024448871612549, -0.26624876260757446, 0.7607421278953552, -1.2176902294158936, 1.0539510250091553, -0.6655125021934509, -0.5837457180023193, -0.16567851603031158, 0.5178366899490356, 1.4065860509872437, 1.4129215478897095, 0.2219710648059845, -0.8504213094711304], [-0.22937525808811188, -0.031123194843530655, -0.055857107043266296, 0.21047967672348022, 0.9611580967903137, 2.234551191329956, -0.1838817149400711, 0.7384316921234131, 0.026146981865167618, -1.6357903480529785, -1.2840211391448975, 0.6176198720932007, -1.4593727588653564, -0.2098316103219986, 1.1876863241195679, 0.6429346799850464, -0.5395729541778564, -0.008836508728563786, -1.2228063344955444, -0.24795305728912354, 0.6503553986549377, -1.4256232976913452, 0.5615832209587097, -0.3153071403503418, -1.2992898225784302, 1.0889893770217896, 0.07803076505661011, -0.31920233368873596, -1.5192937850952148, 0.3481908142566681, -2.215482234954834, 1.1557879447937012, -0.5346298217773438, 0.6584802865982056, 0.6010912656784058, -0.66454017162323, 0.2986738979816437, -0.44424617290496826, -1.1085931062698364, -0.44855883717536926, -1.1262568235397339, -0.228068009018898, 0.1576799601316452, -0.8165335655212402, -0.1820629984140396, 1.820281982421875, 0.5505698919296265, -0.8430696725845337, 0.5476015210151672, 0.0947728306055069, 1.4444105625152588, 0.28812462091445923, -0.6209930777549744, 0.11789995431900024, 0.02130158804357052, -1.2251816987991333, 1.2836378812789917, -1.7257939577102661, 0.3767150938510895, -0.6631196141242981, 0.32816246151924133, -0.9691484570503235, 1.8824623823165894, -0.362898588180542, -0.4394547939300537, -0.6283766627311707, -0.14986498653888702, -1.0145827531814575, -0.7844410538673401, -1.393104910850525, 0.27019575238227844, -0.27518758177757263, 0.917082667350769, 2.478748083114624, -1.3233318328857422, -1.1581088304519653, 0.8870998620986938, -0.13191869854927063, 1.3562253713607788, 1.6705775260925293, -1.791121244430542, 0.30595701932907104, 1.5744013786315918, -0.6627724170684814, 0.4029735326766968, -0.6727270483970642, -0.1779399961233139, 1.357326865196228, 0.5597015023231506, 0.4183037281036377, 1.0682486295700073, -2.447343349456787, -0.37726372480392456, 2.046715497970581, -0.02888103947043419, -0.1950983703136444, -0.19311657547950745, -0.07306718081235886, -1.7268187999725342, 0.5532025694847107, -1.332375168800354, 0.08652910590171814, -1.790600061416626, -0.949612557888031, 0.7657254338264465, 0.003186602611094713, -1.3498494625091553, -0.5259572863578796, -0.3331679105758667, 1.8878159523010254, 1.1131558418273926, 1.1645824909210205, -1.8494517803192139, 0.1464487612247467, 0.6643325090408325, 1.184160590171814, -0.7967758178710938, 0.5356060266494751, -0.69825279712677, -1.3194807767868042, -0.05202692002058029, -0.20720352232456207, 0.9486193656921387, -1.7991572618484497, 1.4323859214782715, -0.67196124792099, 0.3060544431209564, 0.1665516346693039], [-0.7495365738868713, 0.289816677570343, 0.2763743996620178, -0.9167601466178894, -0.06960836797952652, 0.6875860095024109, 2.0262627601623535, 0.023069560527801514, 0.00592839764431119, -0.011107479222118855, 1.1628835201263428, 1.4627048969268799, -2.0138039588928223, 0.6218153834342957, 0.16448424756526947, 0.8437315225601196, 0.9070429801940918, 0.48382502794265747, 1.1046441793441772, -1.039840817451477, 0.4912317991256714, -0.009657252579927444, 1.0827234983444214, 2.7579500675201416, 0.08270761370658875, -1.5380656719207764, -0.8046492338180542, -0.1735735386610031, 0.6292732357978821, -0.36397555470466614, -0.5309457182884216, -0.3532108962535858, 1.4162640571594238, 2.009626865386963, 0.7831499576568604, -0.77842777967453, 0.13404904305934906, 0.038340967148542404, -0.6491419076919556, -0.667774498462677, 0.8871328234672546, 0.2961488664150238, 0.7498253583908081, -0.29852423071861267, 0.5741457939147949, 0.9322463870048523, -0.5296760201454163, 0.8260870575904846, -0.25895535945892334, 0.26911741495132446, -1.0507246255874634, 0.5621358156204224, -0.6125735640525818, -0.19398662447929382, 1.7742259502410889, 2.5579352378845215, -0.9168949723243713, -0.7850245237350464, 0.30661794543266296, -0.401475191116333, 0.08200126886367798, 0.37024542689323425, 0.31795021891593933, 1.4296648502349854, -1.52814781665802, -0.4767799377441406, 0.0327485166490078, 2.0761234760284424, 0.75562584400177, -0.6791477799415588, -1.1953787803649902, 1.5224689245224, -0.31244325637817383, -0.33280161023139954, 0.01546926237642765, -2.8689053058624268, -0.6836438179016113, 2.0227744579315186, -0.3913925290107727, 1.5883028507232666, 2.152780771255493, -0.32772019505500793, 0.9175805449485779, 0.21361036598682404, -0.4049994945526123, -0.9937410950660706, -1.0420360565185547, 1.1010117530822754, -0.4243997037410736, -2.5697004795074463, -0.7223507165908813, -0.9854602813720703, -1.11403489112854, 1.3373929262161255, 2.707960605621338, 0.7765424847602844, -1.5810515880584717, 0.7110681533813477, 1.0449658632278442, -0.7678543329238892, 0.7609748840332031, -0.8403792381286621, -0.9949334859848022, -0.06835244596004486, 0.2205619215965271, -0.16469286382198334, -1.8071871995925903, 1.0431060791015625, -0.2586037516593933, -1.1233512163162231, -0.4809536337852478, -1.8237900733947754, 1.0268901586532593, -0.7550472021102905, 1.1682747602462769, -0.02460523135960102, -1.3756828308105469, 1.0934780836105347, 1.5033140182495117, 0.033240098506212234, -0.5758990049362183, 2.6671082973480225, 0.7290412783622742, 1.6590434312820435, 1.4265378713607788, -0.3463164269924164, -0.05815422534942627, 0.726638674736023]], [[0.8850992321968079, 0.5286805033683777, 0.8689478635787964, 0.6420431137084961, 1.7648595571517944, -0.735386848449707, 1.6445454359054565, -0.579990804195404, 0.1974976509809494, 0.16334566473960876, -1.8110263347625732, 0.13893382251262665, 0.24285238981246948, -0.366366446018219, -1.3544626235961914, -0.2713579833507538, -1.218735694885254, -0.5570309162139893, 0.9608163237571716, -2.214750289916992, 0.13371652364730835, -1.6347789764404297, -0.5725126266479492, 0.6523431539535522, 1.591626524925232, 0.6647078394889832, 0.4055003523826599, 1.2829071283340454, -0.05903242155909538, 0.07644271850585938, 0.14983469247817993, 0.776716411113739, -0.1694176346063614, 0.3864346146583557, -0.8065412044525146, -1.6711373329162598, 1.7495296001434326, -1.3146501779556274, -3.0009641647338867, -1.1378190517425537, 0.28087979555130005, 0.9018028974533081, 0.6937527060508728, 1.3810782432556152, 0.957517147064209, -0.6387594938278198, 0.2694505751132965, 1.130252718925476, -1.3691225051879883, -1.3073103427886963, -0.836344838142395, 0.16257154941558838, -0.7420853972434998, 0.5948907732963562, -0.562741756439209, -0.6185738444328308, -0.314914345741272, -1.345479130744934, 0.32048237323760986, 0.9454467296600342, 1.3995018005371094, -0.12250054627656937, -1.024654746055603, -0.23642733693122864, 0.2143067568540573, -2.202511787414551, 0.09987431019544601, 1.5684130191802979, 0.8065282702445984, 0.1363227665424347, -0.4383928179740906, 0.9147171378135681, -1.2041566371917725, 1.0572199821472168, -1.362446904182434, -0.7296232581138611, 1.4908345937728882, 0.12084552645683289, -0.11519819498062134, -0.6495336890220642, -1.0253666639328003, -0.2945949137210846, 0.4364413917064667, 0.7392673492431641, -0.3113986551761627, 0.14095915853977203, 0.09290292859077454, 1.5781073570251465, 0.5728262066841125, 0.8123517632484436, 0.8339624404907227, 0.2691124677658081, 2.2863922119140625, -0.9091733694076538, 0.634943425655365, -0.2948901057243347, -1.3943302631378174, 0.0529462993144989, 0.13447777926921844, -0.3996135890483856, 1.8692270517349243, -1.468569278717041, 0.6597849726676941, -0.3745601773262024, -2.0715439319610596, -2.0029969215393066, -0.7555241584777832, -1.0891473293304443, -1.3767297267913818, 1.1750922203063965, 1.3370405435562134, -0.8820752501487732, -0.18145258724689484, 0.8347383737564087, 0.6347784399986267, 1.111886978149414, 0.053487103432416916, -0.756413996219635, -2.326230049133301, -0.7936902046203613, 1.0184942483901978, 0.03123595006763935, 0.01445510983467102, -0.2619287073612213, -0.28793373703956604, 1.2162154912948608, -0.3412877023220062, 0.38045617938041687], [-0.40247076749801636, 0.7050389051437378, 0.4647284746170044, -0.049620650708675385, 1.2769240140914917, 0.4174407422542572, 0.5267298221588135, -2.204819440841675, -0.5735340714454651, 0.6535641551017761, -0.5120236277580261, 0.9148529767990112, 1.1265714168548584, 1.144731044769287, -1.1254204511642456, 1.2779998779296875, -0.8648334741592407, 0.7678146362304688, 0.5801665186882019, 0.03062237985432148, -0.2739723324775696, 0.4389873445034027, -1.84566330909729, -0.43597522377967834, 0.5560891032218933, -0.4010792374610901, 0.36865130066871643, -0.000985527760349214, -0.06966947764158249, -0.05920213460922241, 0.154587522149086, 0.25181078910827637, 0.2309584766626358, -0.9223660230636597, -0.5635623931884766, -1.512200951576233, 0.6306386590003967, -0.24386988580226898, 0.007616135757416487, 2.2169225215911865, 1.041234016418457, 0.0041619581170380116, -0.20398280024528503, -1.6066275835037231, -0.027625560760498047, 0.6254021525382996, -0.7580428123474121, 0.9075940251350403, -0.2698226571083069, 0.6127871870994568, -0.04539920389652252, -1.7614898681640625, -1.5379287004470825, 0.3349890112876892, 1.0188586711883545, 0.028669677674770355, 0.8001497387886047, -0.08129678666591644, -0.5846346616744995, -0.2921786904335022, -0.8262592554092407, -1.010221242904663, -0.3576492369174957, -0.4041513502597809, 0.25075846910476685, -0.9899715185165405, 0.22246579825878143, 0.21760232746601105, 0.4136122465133667, 0.40055033564567566, 0.2468847632408142, -1.3847085237503052, -1.2263306379318237, 0.8893405199050903, -1.3161096572875977, 1.5707083940505981, 0.407284677028656, -0.8047165870666504, -0.19513505697250366, -0.08807256072759628, -0.6483258605003357, 1.1451613903045654, -0.33087921142578125, -0.27948594093322754, 0.5241959691047668, 2.2234792709350586, -0.7710702419281006, -0.9341614842414856, 1.0588068962097168, -1.2375930547714233, 1.2121022939682007, -0.413492888212204, 0.3853154480457306, -1.5793591737747192, -0.4587918519973755, 1.656246304512024, 0.09201611578464508, -1.1780543327331543, 0.6444993019104004, -0.6662448048591614, 0.7964503765106201, 1.169634461402893, 0.8636000752449036, -0.3072330355644226, 0.7376287579536438, -1.7842183113098145, 0.41921186447143555, 0.9732400178909302, -0.6862853765487671, -1.2421895265579224, 0.6263681650161743, -2.0457684993743896, 0.8709143996238708, 0.14546045660972595, 0.7827115654945374, -0.7669579982757568, 0.15098097920417786, -1.154517650604248, 0.33466285467147827, 0.014339687302708626, 0.3082050681114197, -1.2653433084487915, -0.17103081941604614, -0.6261332035064697, -0.41057440638542175, -1.3452434539794922, 0.4999348223209381, -1.220991611480713], [0.17983640730381012, 0.8114235997200012, -1.4515953063964844, -2.1070878505706787, -1.2618211507797241, -0.30121615529060364, 1.0022579431533813, -0.821405291557312, -1.8701058626174927, -0.8224936127662659, -2.4057204723358154, 0.012272097170352936, 0.3681175708770752, 0.7624445557594299, -1.0361039638519287, -1.4063141345977783, 0.1920076161623001, -0.23096996545791626, 0.8415234088897705, -0.6539611220359802, 0.48613542318344116, -1.114633560180664, -0.7139500379562378, -1.7391771078109741, -0.15570758283138275, 1.1677656173706055, 0.7230793833732605, -0.22793720662593842, 0.3224608600139618, -0.7986159324645996, -0.9822906851768494, 1.762001872062683, 0.1745402067899704, -0.018292861059308052, 0.043459076434373856, -0.38777589797973633, 0.889518141746521, -0.05537937954068184, -0.6374800205230713, -0.02199222333729267, -0.8874300122261047, -1.526793360710144, 0.7749478816986084, 0.03850467503070831, 0.6600638031959534, 1.7167093753814697, 0.7098115682601929, 1.5603337287902832, -1.9401814937591553, -0.7014212012290955, -0.9961686134338379, -0.8830171823501587, 1.8845720291137695, 0.5243831872940063, 0.5638307929039001, 0.8511644601821899, -0.32742810249328613, -1.054770588874817, -0.16179881989955902, -0.9942417740821838, -0.18880894780158997, 1.187142014503479, -1.302049994468689, -0.5954400897026062, -0.9220054149627686, -0.6142706871032715, 0.4932188391685486, 1.0447465181350708, -1.0160529613494873, 0.10138825327157974, 1.8177322149276733, -1.496119737625122, 0.1621619313955307, -0.47443515062332153, -0.15886570513248444, -0.8960151672363281, 1.6058201789855957, -1.0250945091247559, -0.6527813076972961, -1.2043043375015259, 0.11198198795318604, 1.1264336109161377, 1.3049622774124146, -0.8426914215087891, -1.196976900100708, 0.5191958546638489, 1.7184816598892212, 1.0095077753067017, 1.5038951635360718, -0.703671395778656, 0.0931754782795906, -0.22415056824684143, -1.9595880508422852, 0.3059004247188568, 0.9074344635009766, -0.5886135101318359, -0.2764226496219635, 0.6603103280067444, 0.30912351608276367, 2.344944477081299, -0.0005100545822642744, -0.6466380953788757, 0.36346468329429626, 0.14833472669124603, -0.3706783652305603, -0.7882305383682251, -0.19979873299598694, 2.1047918796539307, 1.9190804958343506, 1.4236849546432495, 1.338339924812317, -0.15168891847133636, 0.8359988331794739, -0.5811471939086914, -1.363901972770691, 1.7471745014190674, -2.2048985958099365, 1.1111685037612915, 0.8812979459762573, -0.02938631922006607, -0.7913244366645813, 0.11986186355352402, -0.667799174785614, 1.7724382877349854, 0.1768869310617447, 0.1263953298330307, -0.5378066897392273, -0.5501641035079956], [0.45145881175994873, -0.8024853467941284, -1.6812547445297241, 1.4585628509521484, -0.10748164355754852, 1.35025954246521, 1.4749056100845337, -0.6373506188392639, -0.8065494894981384, -1.7959064245224, -1.730602502822876, 0.3002772927284241, -0.7585769891738892, -2.1173667907714844, 0.7637004852294922, -0.34446120262145996, 0.46638724207878113, -0.41647541522979736, 2.587184429168701, -0.050271324813365936, -1.2465260028839111, 1.201189398765564, 0.7284986972808838, -1.9742497205734253, -0.38716861605644226, -1.6272475719451904, -0.6020013689994812, -0.3719242811203003, 0.9112023711204529, -1.2853997945785522, 0.20485010743141174, 0.8033133149147034, 1.4952749013900757, -0.9044205546379089, 0.5271101593971252, -0.7754635810852051, 1.3687843084335327, -0.0036056083627045155, -1.1203325986862183, 0.3127640187740326, 0.6927917003631592, 0.4447643756866455, -1.9060312509536743, 0.2291562557220459, 1.8701744079589844, -0.2195487767457962, -0.08215557783842087, 1.8925105333328247, -0.5629058480262756, -1.190555453300476, 0.5745882987976074, -0.04991242662072182, 0.39264827966690063, -0.2513982951641083, -0.5095597505569458, -0.10712248086929321, 1.468181848526001, 1.146549105644226, -0.12625627219676971, 0.9579464793205261, 0.5413140654563904, -1.39197838306427, -0.8036931753158569, -0.1971176564693451, 0.7561429142951965, 1.3407304286956787, 0.5997816324234009, 1.1086108684539795, 0.7366309762001038, 1.4530903100967407, -1.6444681882858276, -0.3796483874320984, 0.7469433546066284, -0.13452161848545074, -1.5073878765106201, 0.38131964206695557, -1.0355738401412964, -0.08510482311248779, -0.7739015817642212, -0.03678840771317482, -1.4780045747756958, -1.0423486232757568, 1.4537736177444458, 0.7113146781921387, 1.0213874578475952, 0.3301173448562622, 0.4186212122440338, -0.07209673523902893, 1.5687596797943115, 0.7917729616165161, -0.011955322697758675, 0.14067262411117554, -0.3797004818916321, -1.0572023391723633, -0.02816089428961277, 0.27718403935432434, 0.3290617763996124, -1.179874300956726, -0.47261905670166016, -0.951472818851471, 1.5061192512512207, 0.7455956339836121, -1.809942364692688, 0.8851211071014404, -1.0223592519760132, -0.14406505227088928, 0.048103757202625275, -0.044191569089889526, -0.6082586646080017, 0.18686921894550323, -0.5717470645904541, -0.4111649692058563, 0.8339163661003113, 2.3267478942871094, -0.22625626623630524, 0.885214626789093, -0.9756794571876526, 1.0637215375900269, -0.4679264724254608, 1.051301121711731, -0.6749570965766907, -0.8170035481452942, -0.16773714125156403, -2.7379090785980225, 0.7424992322921753, 1.0348167419433594, 0.5960706472396851, 1.4048875570297241], [0.7594674229621887, 1.786660075187683, 1.138571858406067, 0.01088176853954792, 0.28767165541648865, -0.28999581933021545, 0.22159095108509064, -1.8956804275512695, 0.5513370037078857, 2.06270694732666, -0.456340491771698, 0.6766911149024963, -0.11474812775850296, 0.15110397338867188, -1.7537319660186768, 0.02088615857064724, 0.30812227725982666, -2.020993232727051, -0.6529929637908936, 0.7823874950408936, 0.06544281542301178, -0.06281684339046478, -0.4865894317626953, -1.9388997554779053, -0.8672545552253723, -1.1062870025634766, 0.21590711176395416, 0.6110736131668091, -0.6514177322387695, 0.4443950653076172, -0.06251504272222519, 0.9349879622459412, -0.5681464076042175, -1.3417574167251587, -1.1788831949234009, 0.8973009586334229, 0.6837441325187683, -0.8530069589614868, -1.5126897096633911, 0.2358139157295227, 1.0865315198898315, -0.11802871525287628, -1.049816370010376, -0.20066829025745392, 1.0529519319534302, -2.366326332092285, 0.28431108593940735, 0.0779903456568718, -1.225426197052002, -0.10350138694047928, 1.2117741107940674, 2.57564640045166, 0.5209547877311707, 1.0155824422836304, -1.2124320268630981, 0.5010269284248352, 0.10217426717281342, 0.724388062953949, -1.7343593835830688, -2.002938747406006, 0.17932990193367004, 1.9069582223892212, -0.5787001848220825, 1.6415811777114868, -0.3282712697982788, -1.235344409942627, 0.33217546343803406, 1.191973090171814, 0.071429044008255, 0.06892271339893341, 0.03828338906168938, -2.2654497623443604, -0.4758862257003784, -0.2750108540058136, -0.4422568380832672, -1.7058897018432617, -0.815676212310791, 0.2679287791252136, 0.9210711717605591, 1.3208036422729492, -0.837392270565033, -0.14688041806221008, -1.7762070894241333, 0.6810738444328308, 0.5677944421768188, 1.611080288887024, -1.2383232116699219, -1.1011534929275513, -1.6295629739761353, -0.4124632477760315, -0.36909446120262146, 1.012393832206726, -0.4986778795719147, 2.035832166671753, -0.5139064192771912, -1.0818250179290771, 0.6748130917549133, -0.34446170926094055, -0.6447256803512573, -0.8316686749458313, 0.20613059401512146, -0.08014032989740372, 0.8916856050491333, -0.014120583422482014, 1.6291687488555908, 0.5761790871620178, -0.6200485229492188, -1.4335007667541504, -0.44596758484840393, -0.7597429752349854, -1.5525175333023071, -0.31713563203811646, -2.5085082054138184, -0.3961022198200226, 0.1502181887626648, 1.1947599649429321, -0.23220236599445343, 0.8222458958625793, 0.5988042950630188, 0.08933757245540619, 0.5824880599975586, -0.49389883875846863, -0.4065026342868805, 1.1147658824920654, 0.48080363869667053, -0.6029148101806641, 1.0328431129455566, -0.2018447071313858], [-0.4288385510444641, 0.45843952894210815, 1.5147089958190918, 0.09411876648664474, -1.1412711143493652, 0.6168781518936157, -1.7053149938583374, 0.33075806498527527, -0.6011328101158142, 0.5707609057426453, 0.13541820645332336, 0.9665666222572327, 1.1805237531661987, -0.21251657605171204, -0.4097997546195984, 2.0111653804779053, 1.6167649030685425, 1.09841787815094, -1.701141119003296, -2.3221867084503174, 0.25121209025382996, 0.9902663826942444, 0.8965124487876892, -1.300794243812561, -0.5591216087341309, 1.060726523399353, 0.19480657577514648, 0.01631161943078041, 0.8480908870697021, 0.13570889830589294, 0.09770683199167252, -1.6663519144058228, 0.17231886088848114, -0.631532609462738, -0.6122399568557739, -0.05976039916276932, 0.9212742447853088, 1.3985179662704468, 0.37013939023017883, -0.027326062321662903, -0.1449383944272995, 0.4699151813983917, -0.7404915690422058, 0.8411062955856323, 2.5615782737731934, 0.381664514541626, -2.274453639984131, -1.5099289417266846, 1.4066431522369385, -0.28298649191856384, -1.326179027557373, -1.6361724138259888, -1.6058447360992432, 1.4780702590942383, -0.06882055848836899, 0.9883670806884766, 1.4376585483551025, -0.1912011355161667, 2.2557921409606934, 0.9046629071235657, -0.08640112727880478, -3.269139289855957, 0.13835011422634125, -1.6958112716674805, -0.6762158870697021, 0.5310284495353699, 0.734212338924408, 1.406038522720337, 1.2815501689910889, -1.61508309841156, -0.21160465478897095, 2.1203725337982178, 0.2528650760650635, -0.9238028526306152, -1.0045865774154663, -0.5363798141479492, -0.7805497646331787, 1.0116965770721436, -0.31991076469421387, -0.2355232834815979, -1.6584813594818115, -0.3794320225715637, -0.9173640012741089, 0.05334426462650299, 0.8957672715187073, 0.8818550705909729, 0.5294235348701477, -0.5356714129447937, 1.0807099342346191, 1.7496886253356934, -0.5362966060638428, 1.2905925512313843, 0.82881098985672, -1.3738195896148682, 0.8555617332458496, 1.036695957183838, 0.05006558820605278, 0.17389973998069763, 2.120009422302246, -1.3925727605819702, 0.8784825205802917, -0.9639129042625427, 0.11435549706220627, -2.0351853370666504, -0.41536304354667664, -1.7792521715164185, 1.2518224716186523, 0.3827633857727051, 0.47350260615348816, 0.4447956085205078, 1.3297419548034668, 0.3494575619697571, 0.6102651357650757, -1.0905797481536865, 0.6403843760490417, 1.488165259361267, -1.8714735507965088, 1.2954457998275757, 0.7113491296768188, 2.151590585708618, 0.12226110696792603, -1.8738459348678589, 0.38215774297714233, 0.7661396861076355, -0.957943856716156, -0.12947142124176025, -0.8496679663658142, -0.27559998631477356], [-0.5320276021957397, -0.4039452075958252, -0.06054956838488579, -0.3662797510623932, 1.7597440481185913, 0.49131718277931213, -0.6166833639144897, -2.918527603149414, 0.47681373357772827, -0.8635560274124146, 0.8852725625038147, -0.8140056729316711, 0.3983913064002991, 1.8599250316619873, -1.3584811687469482, -1.2505314350128174, 0.5714081525802612, 2.8022704124450684, 1.9036530256271362, -2.1746463775634766, 1.4508812427520752, -0.7335827946662903, 0.3000681400299072, 1.231185793876648, 0.5455114245414734, -0.6231940388679504, -0.8872793912887573, -0.2717525362968445, -0.5672298669815063, -0.5130577683448792, 0.7535818815231323, 0.9354464411735535, -0.3138478994369507, 0.6964871883392334, -1.3166966438293457, -1.3654217720031738, -0.8776363730430603, -0.2218049317598343, -1.168105959892273, -0.21168194711208344, 0.9215131998062134, 1.2909841537475586, 0.5533900856971741, -1.3006854057312012, 0.25543588399887085, -0.2964540719985962, -0.20801451802253723, 1.3411099910736084, -1.5189926624298096, -0.6277002096176147, 0.20655900239944458, 2.38588285446167, 0.7312127351760864, 0.06654275953769684, 0.8378162384033203, -0.4700811505317688, -0.9202622175216675, -1.3195226192474365, -0.12911179661750793, 0.48397818207740784, -1.094059944152832, 0.5973963737487793, 0.19152843952178955, -0.669238805770874, 0.2662239372730255, -0.6062288880348206, 0.7678468823432922, -1.098924994468689, 0.8170390129089355, -1.7462960481643677, -0.6226941347122192, 0.6962659955024719, -0.45905938744544983, -1.0121132135391235, 0.694911777973175, -0.18924923241138458, 0.6078631281852722, -0.645259439945221, 1.3598978519439697, -1.0682239532470703, -0.5829082131385803, 0.26736411452293396, -1.2090855836868286, 0.10423848032951355, 0.11041120439767838, 0.2240578681230545, 1.3329792022705078, -0.8841721415519714, 0.8141810297966003, 0.11464507132768631, -1.40066397190094, 0.7392211556434631, 1.553688883781433, 1.3246582746505737, 1.1329412460327148, 1.0946582555770874, 0.08789772540330887, 0.15456169843673706, 0.3372528851032257, 0.4191930294036865, 0.31826457381248474, -0.46316125988960266, -0.7159301042556763, -1.2101374864578247, 0.2643514573574066, 0.04037405550479889, 0.13053685426712036, -1.1712521314620972, 0.048580557107925415, 0.5260931849479675, -3.3188915252685547, -0.6736699938774109, -0.1684463620185852, 0.36893120408058167, -0.18165870010852814, 0.30145689845085144, -1.2253310680389404, 0.5391043424606323, 0.27389055490493774, -0.6573556065559387, -0.48489895462989807, 0.004275510087609291, 0.27043697237968445, 1.007715106010437, 0.5526835918426514, -1.212685465812683, -1.109268307685852, -0.21201568841934204], [1.1388769149780273, 0.3066249489784241, 0.2993684411048889, 0.10624997317790985, -1.0134090185165405, -1.0467172861099243, 2.416402578353882, 1.470818042755127, -0.44078195095062256, -1.442833662033081, -1.4790321588516235, -0.5954346060752869, -0.4655289053916931, 0.8334267735481262, -0.6620232462882996, -0.20410507917404175, -1.2586126327514648, 3.506387233734131, 0.046615373343229294, 1.221761703491211, -0.17152881622314453, 1.163325548171997, -0.5714741945266724, -2.584460735321045, 0.36169761419296265, -0.8166435360908508, -0.9954070448875427, 0.3497762382030487, -1.5969884395599365, 0.06136832386255264, -0.3927527666091919, -1.5584678649902344, 0.7042555809020996, 0.23851804435253143, 1.4036628007888794, -0.02817625179886818, 0.11402624845504761, -1.89210844039917, 1.37606942653656, -0.22022350132465363, 0.3023419976234436, -0.6486893892288208, 0.4816665053367615, -0.2966707944869995, 0.35033610463142395, -0.42465487122535706, -1.143200397491455, 1.4351732730865479, -0.029986711218953133, 0.07147163152694702, -0.34661704301834106, -0.873610258102417, -1.0331193208694458, 1.1282049417495728, -0.26865577697753906, -0.7375953793525696, 0.7437908053398132, 0.8069785237312317, 0.6325610280036926, -0.7955195307731628, -0.6191261410713196, 0.4371333420276642, 0.3826337158679962, 0.91144198179245, 1.9268232583999634, -0.31386440992355347, 1.7351961135864258, -0.0739247277379036, 1.5652360916137695, 0.10011209547519684, 0.15336526930332184, 0.1952826827764511, 1.3614983558654785, -0.6449835896492004, 1.0142349004745483, -0.07607141882181168, -0.8742741942405701, 0.06755862385034561, 1.041377305984497, -0.23429091274738312, 0.4221493601799011, -0.14561833441257477, -0.6930682063102722, -0.6471680998802185, 1.4133045673370361, 1.5779446363449097, 1.142539381980896, 1.8553465604782104, -0.5566708445549011, -0.7863161563873291, -0.7233871221542358, -0.1911851018667221, 1.1216504573822021, 0.9940618276596069, -2.4914517402648926, -1.0547841787338257, -0.5410569310188293, -0.3845423460006714, -1.0406978130340576, -1.5494954586029053, 1.2590445280075073, 0.6404863595962524, -1.5350135564804077, 0.31798890233039856, 1.168419361114502, -0.8221219182014465, 0.12464167922735214, -0.7128528356552124, 0.0892946869134903, -0.1644621640443802, -0.24298183619976044, 0.4180961847305298, -0.30708369612693787, -0.07467953115701675, 0.7580147981643677, 0.6284489631652832, -1.2611565589904785, -1.1896363496780396, 3.240811824798584, 1.4183253049850464, 2.24988055229187, -0.818076491355896, 0.43611985445022583, 1.3417823314666748, -0.3734663724899292, 1.68382728099823, -1.4660953283309937, -0.8826000690460205]], [[1.2009174823760986, -0.8675821423530579, 0.5761470794677734, -1.5438376665115356, 0.13041968643665314, 1.1756283044815063, -0.07015308737754822, -0.9712795615196228, -0.5581849217414856, 1.2248320579528809, -1.6253644227981567, 0.9138697385787964, -0.5298187732696533, 0.7297332286834717, 0.6150838732719421, 0.02559727616608143, -1.2697839736938477, -0.9979952573776245, 1.8046444654464722, -0.9193978309631348, -0.15768858790397644, -0.6793801784515381, 3.429375171661377, -0.4819544553756714, 0.08249883353710175, -0.08002568781375885, -1.9676878452301025, -0.276399701833725, -1.612253189086914, 1.00288987159729, -0.7603033781051636, -0.08130095899105072, 1.6039917469024658, -0.7620193958282471, 0.759360671043396, 0.10418631881475449, 0.5307202339172363, 2.0691871643066406, -2.352811574935913, -1.6262145042419434, -0.474073588848114, 1.1704527139663696, -1.4309489727020264, -2.27817702293396, 0.7297410368919373, 0.16511647403240204, -0.23822341859340668, 0.685783863067627, -0.8327467441558838, -0.392826646566391, 0.6070143580436707, 0.5177775025367737, -0.20783883333206177, 1.1301568746566772, 1.3293724060058594, 1.2250877618789673, -1.6178979873657227, -1.2262980937957764, 0.026659665629267693, -2.1713104248046875, 0.7323548793792725, 0.6672484278678894, 0.6036314368247986, 0.8190091252326965, -0.2512771189212799, -1.880602240562439, 0.13772521913051605, -0.13254252076148987, 0.35890379548072815, 1.0807875394821167, 2.375208854675293, -0.9189822673797607, 1.2065191268920898, 0.5342784523963928, -1.5757519006729126, 0.09851465374231339, 0.3029076159000397, 0.38041242957115173, -0.5346462726593018, 0.9276013970375061, -0.21714971959590912, -1.9598071575164795, 0.2500750422477722, 0.8031461834907532, 0.7837749719619751, 0.8447240591049194, 0.8667970895767212, -0.16033761203289032, -0.29822131991386414, 0.16424106061458588, 0.19231662154197693, -0.5032452940940857, 0.1347983032464981, 1.5500938892364502, -1.1364200115203857, -0.9340903162956238, -0.020710736513137817, 0.20261283218860626, -2.1525042057037354, -1.0938067436218262, -1.0149577856063843, -0.5457888245582581, -2.238477945327759, -0.5336954593658447, -1.8661648035049438, 0.1773684024810791, -2.085038661956787, 0.42166200280189514, 0.1214689388871193, 1.8375486135482788, -0.507271945476532, -0.9809824228286743, 1.2821522951126099, -0.41394558548927307, 0.4088127911090851, 0.7285826802253723, -0.14481714367866516, -1.8763082027435303, 0.19650082290172577, 0.5974599123001099, -1.8616986274719238, -0.44473907351493835, -0.7675560712814331, 0.46202290058135986, 1.2547240257263184, -1.123530626296997, -0.9099634289741516, -1.090590238571167], [-0.3162306249141693, -0.153779998421669, -0.6334304809570312, -0.7543436288833618, -0.4048815965652466, -1.180226445198059, -0.5195719599723816, -0.12558656930923462, 1.1184167861938477, -0.8712181448936462, -0.005975756794214249, 0.2739439904689789, 2.5689151287078857, -2.022331953048706, -1.160978078842163, 2.813981056213379, 0.8872988224029541, -0.0796402245759964, 0.465017169713974, -0.13754023611545563, -0.6771173477172852, -0.7322629690170288, -0.11137151718139648, 0.9915911555290222, -0.5069049000740051, -0.6790973544120789, -1.1179059743881226, 0.33148059248924255, 0.2804984450340271, -0.8653160333633423, 0.7341504693031311, 1.0645802021026611, -1.171674132347107, 1.5326935052871704, 0.10638201981782913, -1.6604806184768677, 1.2703721523284912, 0.8793841600418091, -0.052906300872564316, -0.5885067582130432, 1.9595874547958374, -0.025638561695814133, -0.6163375377655029, -1.1274738311767578, 2.1538491249084473, 1.0616849660873413, 0.3536049723625183, -0.376246839761734, 0.08328232169151306, 0.8144335746765137, 0.014226991683244705, -0.2836810052394867, -0.8195034861564636, 0.17413592338562012, 0.12717172503471375, 0.6257282495498657, -0.8956537842750549, 0.37559106945991516, 1.1384103298187256, -0.9622553586959839, 0.6782591342926025, -0.6911979913711548, 0.07365411520004272, 1.307323694229126, 0.1499178260564804, -0.2094903588294983, -1.3337208032608032, -0.912819504737854, 0.3111427128314972, 0.6047066450119019, -0.9328791499137878, 1.210476040840149, 1.2114627361297607, -1.2636362314224243, -0.6609982252120972, 0.3935670256614685, 0.36544302105903625, 0.2592214345932007, 0.04050139710307121, 0.2743974030017853, -2.265773296356201, -0.864595353603363, -0.1328916698694229, 0.2802596092224121, 1.1408061981201172, -0.6186432838439941, 1.1144304275512695, -1.2292299270629883, -0.9453728199005127, -0.21207237243652344, -0.4707627296447754, -1.4146075248718262, -0.2918684184551239, -0.3386579155921936, -0.668483316898346, -0.7632165551185608, -0.4369839131832123, -0.2773902416229248, -0.043466147035360336, 2.299128770828247, -1.089271068572998, 0.6918079853057861, -0.47460582852363586, 0.9776877760887146, 0.49153655767440796, 1.1495614051818848, 0.42064639925956726, -1.060232162475586, -0.6271965503692627, 0.49753713607788086, 0.8545935750007629, -2.184860944747925, 1.3475441932678223, -0.12797407805919647, -0.6999762654304504, -1.9704753160476685, 0.41210484504699707, -0.09231330454349518, -1.4314512014389038, 0.7615250945091248, -0.9078361988067627, -0.08025650680065155, -1.1464570760726929, 2.015545606613159, -1.0776625871658325, -0.4522804915904999, 1.7407945394515991, 1.1346896886825562], [0.30584579706192017, -0.007461026310920715, -0.15187591314315796, -0.9203122854232788, 0.8714451789855957, -0.26580968499183655, 0.4331902265548706, 1.0272241830825806, 0.8711145520210266, -1.217873454093933, -0.6045340299606323, -0.8466005325317383, 0.8323954343795776, 0.18038463592529297, -0.6310415267944336, 0.765936553478241, 2.694854259490967, 1.7331289052963257, -1.1504508256912231, 1.6655296087265015, -0.708206832408905, -0.23735769093036652, 0.48879700899124146, -1.686225175857544, 1.953282356262207, -0.7530363202095032, 2.202134609222412, 1.2956572771072388, -1.0931119918823242, 0.536541223526001, -0.8579248785972595, 0.7208529710769653, -1.0500729084014893, 0.37169548869132996, -0.8736138939857483, -0.8225072026252747, 0.38437607884407043, 0.4374261498451233, 1.4364160299301147, 1.0008907318115234, -0.45305293798446655, -0.11148551106452942, -0.03147326037287712, -0.5620281100273132, 0.46987423300743103, -0.338703989982605, 1.9937896728515625, 0.5770948529243469, 0.24293579161167145, -0.15325886011123657, 0.15152998268604279, 1.5844680070877075, 0.5014476776123047, 0.8930550217628479, -0.10503075271844864, 0.6793599128723145, 0.3608526885509491, 0.033380020409822464, -1.9837439060211182, -0.08006632328033447, 0.21340319514274597, -0.4521769881248474, -0.3762548565864563, -1.0310297012329102, 0.6269415020942688, -0.9046062231063843, -0.6855618357658386, -2.0321431159973145, 2.294665575027466, -0.5553957223892212, 1.0090137720108032, -0.27468135952949524, -1.5257123708724976, 0.4571525454521179, 0.021093010902404785, 0.38237056136131287, 1.2831381559371948, -1.0095797777175903, 0.039683423936367035, -0.12779909372329712, 1.8338342905044556, -0.09990805387496948, -0.7889894843101501, -1.0369271039962769, 0.4779787063598633, -1.659118890762329, -0.05115335434675217, -1.3596270084381104, 1.1274209022521973, -0.9098422527313232, -0.6922556161880493, 0.7134091258049011, -0.5137969255447388, 0.43538784980773926, -1.6836339235305786, 1.2923402786254883, 0.5723896026611328, -0.8455038070678711, 0.9484691023826599, 0.3517141044139862, 0.11680921167135239, 1.059810996055603, 1.3949559926986694, 0.46666815876960754, 0.7666693925857544, -0.1401194930076599, -1.659545660018921, -0.6108800768852234, 0.3854672908782959, 1.181225299835205, -1.8225820064544678, -1.5757067203521729, -0.12668032944202423, -1.722712516784668, 0.6945755481719971, 0.24767623841762543, 1.3177595138549805, -0.3462464213371277, -1.2548654079437256, -1.3464571237564087, 1.374668002128601, -2.6792943477630615, 1.2267463207244873, 0.9651469588279724, 0.9390618801116943, -1.4349029064178467, 1.6203389167785645, -0.5792756080627441], [-2.3051929473876953, -1.4560964107513428, -1.0779153108596802, 1.164103388786316, 0.02138538472354412, -0.11996650695800781, -1.4811015129089355, 0.838236927986145, -0.9641114473342896, -0.20235519111156464, -0.6225857138633728, 0.19848504662513733, 0.9965627789497375, 0.7097697257995605, 1.2002344131469727, -0.2750072479248047, -0.4197622239589691, 1.3132116794586182, -0.2191816121339798, -0.5920640230178833, -0.056938380002975464, 0.1177317425608635, -0.8100337982177734, 0.06853239983320236, -0.6694826483726501, 0.4178226888179779, -0.2883361577987671, 0.30620095133781433, -2.9418785572052, -0.21999506652355194, 0.2678956091403961, 0.09022965282201767, -0.6262716054916382, 1.446523666381836, -0.6579839587211609, -0.05543730407953262, 0.535577118396759, -1.6557503938674927, -1.1376729011535645, -2.4067721366882324, 0.6179801821708679, -0.5668403506278992, 0.10256725549697876, 0.4025351405143738, 1.0775103569030762, 0.6373171806335449, 0.5484556555747986, -0.21673329174518585, -2.748528242111206, -1.1657142639160156, 0.5388422012329102, 0.4698283076286316, -1.0898761749267578, -0.29789310693740845, 0.7363632321357727, -0.08722571283578873, 0.26778092980384827, -0.9804615378379822, 0.10660941153764725, -1.0125620365142822, -0.8239621520042419, -0.572031557559967, -1.8432955741882324, 0.3928402364253998, -0.24854564666748047, -0.9520772099494934, 0.49376925826072693, 0.4925670623779297, -0.784247100353241, -0.48933374881744385, 0.97205650806427, 0.166947603225708, 0.20912335813045502, 0.595085620880127, 0.6119389533996582, -0.6142869591712952, 0.45371997356414795, -0.7399221658706665, 1.0864964723587036, 0.053680699318647385, -1.0923261642456055, 0.16237953305244446, 1.1297240257263184, 0.7257342338562012, -1.5039173364639282, 1.2105926275253296, 0.4516913890838623, -2.388582229614258, 1.0182701349258423, 1.401523470878601, 0.6959302425384521, -0.358316034078598, 1.5927573442459106, 1.0935629606246948, 0.3783894181251526, -1.2105025053024292, 1.0965715646743774, -0.6170107126235962, -0.2915710210800171, -0.753706693649292, 0.5233649611473083, 0.43334880471229553, -0.2918502688407898, 1.9451040029525757, 1.4762682914733887, 1.5027996301651, 1.003888726234436, 1.3792093992233276, -0.38435617089271545, -0.5448910593986511, -0.6322416067123413, -0.9641347527503967, 0.3837394118309021, -0.8618155717849731, 0.863814651966095, -0.6948574781417847, -1.3303884267807007, -0.2057233303785324, 1.6647721529006958, -1.9279091358184814, 0.011567054316401482, 0.8335432410240173, 1.5008957386016846, -0.28824248909950256, -0.509072482585907, 1.4617738723754883, -1.1305574178695679, 1.5530204772949219], [0.36317208409309387, 0.32020747661590576, -1.1683802604675293, 2.20743989944458, -0.5186344981193542, 1.0462360382080078, -0.06604904681444168, -1.404363989830017, -0.6938921213150024, -0.5711743235588074, 0.5076311826705933, 0.7572576403617859, 0.6534256339073181, 1.5105068683624268, -0.5979573726654053, -0.6698784828186035, 1.297296166419983, -2.4145195484161377, -0.13424062728881836, -0.9456708431243896, 1.7639466524124146, -0.11871735751628876, -0.47528284788131714, 0.1511947065591812, -0.6995465755462646, -0.3964887261390686, -0.04588424414396286, -1.88727605342865, -0.9980972409248352, -0.25748905539512634, -1.0606743097305298, -0.6736065149307251, -1.2747465372085571, 0.8539236783981323, 0.7242814302444458, 1.4824904203414917, -0.5041468143463135, -1.9395002126693726, -0.2053806483745575, 0.3232354521751404, -0.04166625812649727, -0.47823089361190796, 2.8321099281311035, -0.5972963571548462, -0.9535308480262756, 0.706425666809082, 0.08249948173761368, -1.5848792791366577, 0.3091321885585785, -0.26573798060417175, 0.09072030335664749, -0.6242237687110901, 0.5786918997764587, -0.8807440996170044, 1.550052523612976, 1.04304039478302, -0.38993123173713684, -0.09044413268566132, 2.5516769886016846, 0.261260449886322, -0.41347452998161316, -0.15643921494483948, 1.2404152154922485, 0.23998293280601501, 0.6802027225494385, 1.4594807624816895, 0.05537432059645653, -1.0292792320251465, 1.244712233543396, -0.9909210801124573, -0.8128551244735718, -1.382880687713623, 0.8349146246910095, 2.1527156829833984, -1.4940320253372192, -2.698326349258423, 0.010051093995571136, -0.5209125876426697, -1.0431702136993408, 1.2507665157318115, -0.5009889602661133, -1.2510724067687988, 1.3619449138641357, 0.1844508796930313, 1.2150510549545288, -0.5246812701225281, 0.1824837028980255, 0.8446640968322754, 0.4001229703426361, -0.46287938952445984, 0.7247135639190674, -1.8567074537277222, -0.6780344247817993, 0.9348533749580383, 0.6976017355918884, -0.7526137828826904, 0.11944538354873657, 0.14362820982933044, 1.0155056715011597, 0.2874428927898407, 0.5630438923835754, -0.9254271388053894, -0.45338699221611023, -0.6093197464942932, 0.6930806040763855, -2.321403741836548, -0.35104674100875854, -0.1211935430765152, 1.294774055480957, -0.2896070182323456, 0.8154228329658508, -0.28891271352767944, -0.12541741132736206, -0.4382709562778473, 0.3753426969051361, -0.340006023645401, 1.5105887651443481, -1.4036602973937988, -0.7006016373634338, 0.0566876195371151, 0.5287162065505981, -1.3883960247039795, -0.19333003461360931, 0.5271168947219849, -0.25882840156555176, -0.9600169658660889, 0.9346587657928467, 1.732216715812683], [-0.44671162962913513, 0.5963215827941895, 0.5446339845657349, 0.28537359833717346, 0.8884989619255066, 1.2865490913391113, -0.37590205669403076, -2.0827622413635254, 0.3318168520927429, -1.499236822128296, 1.3618539571762085, 0.038360659033060074, 1.9213011264801025, 0.2637971043586731, 2.06669545173645, -0.19278591871261597, 1.216695785522461, 0.5308853983879089, 0.9556363821029663, 1.3839104175567627, -0.03892279416322708, -0.15500923991203308, 0.1286378651857376, -0.5208866596221924, -0.5846142768859863, 0.0005931448540650308, -0.2790609896183014, -1.4009653329849243, -1.6038607358932495, -0.9000683426856995, -0.753866970539093, -0.8658168315887451, -0.9276467561721802, 1.908970832824707, -0.8714859485626221, -0.6790194511413574, 0.8271569609642029, -1.0053555965423584, -0.7674031257629395, -2.294611930847168, 0.7530611157417297, -1.118327021598816, 0.041263386607170105, 0.0625811368227005, -1.595603585243225, 1.3754081726074219, -0.26552248001098633, 0.46552687883377075, -0.1494404375553131, 1.7614023685455322, -0.39971446990966797, 0.45219460129737854, -0.2784327566623688, 0.42216575145721436, 0.07277072221040726, 0.5126175880432129, -0.7615494728088379, 0.2944451570510864, 0.7303008437156677, -0.2604246139526367, 1.6688599586486816, -0.36036941409111023, 0.681160569190979, 0.20843608677387238, -1.939094066619873, 0.8139531016349792, -1.3341573476791382, 0.20997263491153717, 0.0479913204908371, -1.0487380027770996, 0.24210014939308167, 1.8060863018035889, 1.3287111520767212, 1.187049150466919, -0.9805488586425781, -0.16382655501365662, 0.4338676333427429, 0.6779592037200928, -0.25233855843544006, 1.7819362878799438, -0.38582512736320496, -0.47218313813209534, 1.3940062522888184, -1.371500849723816, 0.9633839130401611, 1.4876320362091064, -0.14487679302692413, 1.0913150310516357, -1.0569037199020386, 0.17184217274188995, 0.17398414015769958, 0.550950825214386, 1.2954219579696655, -0.7819579243659973, 1.2854113578796387, -0.23778842389583588, -0.3227483034133911, 1.075510025024414, 0.5189072489738464, -0.38626644015312195, -0.08117461949586868, -0.8375645875930786, 1.002989649772644, -0.024875469505786896, 0.6138055920600891, -0.1798483431339264, 0.5375568866729736, -1.7512054443359375, -0.09860214591026306, 1.6310144662857056, 0.9247118234634399, -1.6667768955230713, 0.06895927339792252, 0.49516263604164124, -1.2458852529525757, -1.19141685962677, 0.8449040651321411, 0.5892665386199951, 1.1435014009475708, -1.416942834854126, 0.3367483615875244, -0.9375932216644287, 0.8486459851264954, -0.04537453502416611, -1.0355641841888428, 0.8733097314834595, 1.035853624343872, 1.3495304584503174], [0.8279181122779846, 1.4717466831207275, -0.11481758952140808, -1.091407060623169, 0.6667123436927795, -1.120306134223938, 1.7596728801727295, -0.006034274119883776, 0.0993160679936409, -0.8226660490036011, -1.3057811260223389, -2.738558769226074, -1.1397594213485718, -1.2977309226989746, -0.883438766002655, -0.8295480608940125, -0.14313770830631256, 0.08303488790988922, -0.145940363407135, 0.4991164207458496, 0.8356668949127197, -0.2532272934913635, 1.348040223121643, 0.7272783517837524, -0.08457692712545395, 0.22426089644432068, -0.7179807424545288, 0.3257569670677185, 0.5057400465011597, -1.7359100580215454, -0.9536179304122925, 0.9765622019767761, 0.07403309643268585, 2.197739839553833, 0.8121309876441956, -0.02556357905268669, 0.2734127938747406, -1.0975937843322754, -1.5660290718078613, -1.0681283473968506, -0.6494335532188416, -1.3499118089675903, 0.6114287972450256, -0.6598582863807678, 0.6528928279876709, -0.46340179443359375, -0.4157063663005829, 0.39045843482017517, 0.2093794345855713, -0.01712833344936371, -1.8673392534255981, -0.9411104917526245, 0.5093258023262024, 0.47655314207077026, 0.6542969346046448, 0.02781350165605545, 1.870424509048462, -0.4550381302833557, 0.5945087671279907, 1.6272493600845337, -0.2563038766384125, -1.4076608419418335, 0.2959774136543274, -0.33993735909461975, -0.38104739785194397, -0.5524567365646362, 0.32924672961235046, 2.105499744415283, 1.0599044561386108, 0.8013883233070374, -0.17298981547355652, -0.5666952729225159, -0.9111233353614807, -0.4712781608104706, -1.3172314167022705, -1.718995451927185, 0.657513439655304, -0.13211368024349213, 2.237535238265991, -0.47363972663879395, -1.2713123559951782, 0.09564174711704254, -1.059044361114502, 0.5072075128555298, 0.05826527625322342, -1.3822729587554932, -0.7164633870124817, -0.03503299131989479, 1.4259506464004517, 0.41129565238952637, -0.9437894225120544, 0.4341103136539459, -0.3092705011367798, 1.4926841259002686, -0.18965131044387817, 0.3729722797870636, 0.9984327554702759, 0.4083991050720215, 0.725372850894928, -0.5968465805053711, 0.040041301399469376, 0.41445818543434143, 0.1766815185546875, -0.14635618031024933, -1.5098321437835693, 1.1618759632110596, -0.10851617902517319, -0.9953261613845825, -0.7230258584022522, 0.9678122997283936, -0.325173944234848, 1.4153919219970703, -0.8056815266609192, -0.632988691329956, -0.7276731729507446, -1.22086501121521, 1.1856775283813477, -0.48652851581573486, -2.0147368907928467, 0.9272302389144897, 0.486803263425827, -0.35940074920654297, 0.37560948729515076, -1.9431297779083252, 0.22418396174907684, -0.3405890464782715, -0.20668788254261017, -1.0842300653457642], [-1.13017737865448, -0.3226507604122162, -1.8915313482284546, 1.4417181015014648, -0.5926794409751892, 2.5180439949035645, -1.7564616203308105, -0.5761958956718445, -0.47886037826538086, 0.23314736783504486, -0.3433701694011688, 0.7865749001502991, -0.4421885013580322, -1.0129895210266113, -0.9105768799781799, 0.41600218415260315, -1.86505925655365, -0.37264400720596313, -0.10206270217895508, -1.0086275339126587, -0.15385107696056366, 0.21572306752204895, -0.252156138420105, -0.421860009431839, 0.33715203404426575, -0.10825543105602264, 1.0022404193878174, -0.003612851956859231, -1.0826703310012817, -0.2281009405851364, -0.03448288515210152, 0.6627814769744873, -0.13099631667137146, 0.06588573008775711, 0.5840811729431152, 1.492480754852295, -0.8902900815010071, 0.18108171224594116, -1.3524202108383179, 0.4233979880809784, -0.08851827681064606, 0.26123008131980896, 1.1085950136184692, 0.8376334309577942, 1.5154749155044556, -0.48500269651412964, -0.8237322568893433, -0.15969602763652802, 0.13283811509609222, -1.7463514804840088, 1.132947325706482, 0.18671339750289917, 0.80307537317276, -1.472312569618225, 0.10363627225160599, 0.6344597339630127, -1.5330039262771606, -0.44685256481170654, 0.37091192603111267, -1.3279366493225098, -1.640033483505249, -0.3852396309375763, 2.5661306381225586, -1.2603013515472412, 0.3976423740386963, 0.5379703640937805, -0.4262078106403351, -0.8133429288864136, 0.3902103006839752, 1.3545289039611816, -0.1955554038286209, -0.972843587398529, -0.5633453726768494, 1.0056699514389038, 0.7368043065071106, -0.43045276403427124, 1.0807578563690186, 0.9119852185249329, -1.0602986812591553, 0.4753350019454956, 1.4416234493255615, -0.9894146919250488, -1.0966899394989014, 1.1691919565200806, 0.9016414880752563, 0.06884687393903732, 0.7347795367240906, -0.6251561045646667, 0.09963425993919373, 0.9398435354232788, -0.890680730342865, 0.40825003385543823, -0.7250747680664062, -0.5000740885734558, -0.34788399934768677, 1.3382327556610107, 0.7459318041801453, 0.30585286021232605, -0.7587078213691711, -0.7774474620819092, 0.1416010856628418, -0.19063594937324524, -0.5015101432800293, 0.9082951545715332, 1.3196324110031128, 0.9061993956565857, 0.6970080137252808, 0.5650734305381775, -1.7927321195602417, -0.5840137004852295, -0.25690755248069763, -1.9461393356323242, 1.1289573907852173, -0.08513103425502777, 0.8113396763801575, -0.5634381771087646, 1.4151684045791626, 0.7298493981361389, 0.3472779393196106, -1.975158452987671, -0.5379190444946289, 0.9897561073303223, -0.8789926767349243, 1.1841466426849365, -1.6378426551818848, -1.4461618661880493, -1.5247410535812378, 0.5161603093147278]]], [[[0.14793457090854645, -0.633588969707489, 0.12321283668279648, 0.9565592408180237, 1.8608301877975464, -0.32262659072875977, -0.8997958898544312, 0.5904977321624756, 1.6272974014282227, 0.02249693125486374, 1.5319480895996094, 0.18898913264274597, 1.6118346452713013, 1.0354769229888916, 1.1200318336486816, 0.857553243637085, 0.20291778445243835, -0.6717191934585571, -1.256550908088684, 0.46053943037986755, 0.09872328490018845, 0.5613888502120972, -0.8136757016181946, -0.7240001559257507, 1.8337594270706177, -1.2765815258026123, 1.5460336208343506, 1.4169145822525024, 0.6859226822853088, -0.21410219371318817, -1.6306803226470947, -1.6482253074645996, 1.3330192565917969, -0.49765506386756897, 1.6768172979354858, -0.2778354287147522, -1.724382996559143, -0.21283844113349915, -1.28813898563385, 1.01883864402771, 0.2074287384748459, -1.0786558389663696, 2.348965644836426, -0.43090713024139404, -0.36655887961387634, 0.5552963614463806, 1.540306806564331, 0.48236095905303955, -1.0615895986557007, 1.1511218547821045, 2.048722982406616, -0.46006688475608826, 1.1497470140457153, 0.4357863664627075, 1.0653787851333618, -0.5039926767349243, 0.4744580090045929, -0.046421751379966736, -0.6189548373222351, 0.24998725950717926, 0.21443292498588562, -1.0407925844192505, 1.195282220840454, -1.6665925979614258, 0.3913528621196747, -0.9499136805534363, -1.424364686012268, 0.11201810091733932, 1.492392659187317, -1.108574390411377, -0.02943381480872631, 0.01547225657850504, -0.9962319731712341, 1.1185885667800903, 1.8052352666854858, 0.49814045429229736, 2.259744882583618, 1.4174258708953857, 0.8011571764945984, 0.4795185327529907, 0.89701908826828, 0.34081751108169556, 0.6415333151817322, -1.1948856115341187, -0.425226092338562, 0.11657311022281647, -0.15437886118888855, -3.0620532035827637, -0.4007043242454529, 1.1173474788665771, -1.3315248489379883, 1.2379456758499146, 0.6288782358169556, -0.738993227481842, 1.2607805728912354, 0.27742481231689453, -0.5227211117744446, -0.2387782633304596, 1.3135846853256226, 0.3454268276691437, -0.22661663591861725, -0.5969099402427673, -1.0926653146743774, -0.41232264041900635, -0.7424405217170715, -0.5981184244155884, -1.1831098794937134, 0.08741626143455505, 0.8154475688934326, 2.041370391845703, 1.2765159606933594, -0.725254237651825, -0.5549282431602478, -1.1208770275115967, -0.9236291646957397, -0.41966503858566284, -0.5945371985435486, -0.2904913127422333, 0.5637875199317932, 0.20499631762504578, -0.5627228021621704, 0.3298323154449463, 0.9931952953338623, -0.8835201263427734, -0.26123642921447754, 1.3210862874984741, -0.5961286425590515, -0.17492204904556274], [0.9816085696220398, -0.5271129608154297, 0.4335893392562866, -1.6074280738830566, -1.7814273834228516, 1.0781493186950684, -0.08418775349855423, 0.3743695318698883, 0.26364412903785706, 0.8396492600440979, 1.22954261302948, 1.2713054418563843, 0.7532607913017273, -0.2538449168205261, -1.1431560516357422, 1.5556740760803223, -0.25561967492103577, -0.7326390147209167, 1.5828338861465454, -0.08407402783632278, 0.5516316294670105, 0.16067533195018768, -0.5243122577667236, -0.12907294929027557, -0.5307212471961975, 0.7498432993888855, 0.60356605052948, -0.2765519618988037, 0.6630931496620178, 0.013684921897947788, -0.8465440273284912, -1.8387705087661743, -0.6129588484764099, -0.5017121434211731, 0.09003148972988129, 0.5938981175422668, 0.7318162322044373, 0.16373994946479797, -0.5568187832832336, -1.0191479921340942, -0.716454803943634, -1.8345451354980469, 1.3187657594680786, -1.819589376449585, 0.33262890577316284, 0.9897860884666443, 2.109612464904785, -1.540290117263794, -1.093387246131897, 1.9906392097473145, -0.1307927817106247, -0.11778603494167328, -1.1110739707946777, -0.7718385457992554, 1.1548070907592773, 0.5629271864891052, 0.6915026903152466, -0.13103406131267548, 0.28843969106674194, 0.8702535629272461, -0.039877742528915405, -0.4614766538143158, 1.068325400352478, -0.37409061193466187, 0.723721444606781, -1.247727394104004, -0.1095585897564888, 0.8601913452148438, 1.0132390260696411, 2.0568528175354004, -1.7022075653076172, 1.3760730028152466, -1.2802482843399048, -0.9052299857139587, -1.3089429140090942, -0.2681325674057007, -0.7253541946411133, 0.3171413540840149, 1.5846885442733765, 0.16607387363910675, 0.7705948948860168, 1.3164544105529785, -1.1373310089111328, 1.2093241214752197, -0.3934611976146698, 0.4263542890548706, -0.0028579742647707462, 2.493727445602417, -2.6943717002868652, 0.38280603289604187, -1.1178945302963257, 0.3951108455657959, 0.02042256109416485, -0.5331034064292908, 1.128534197807312, 0.37426015734672546, -0.16220679879188538, 0.22240734100341797, -0.38541606068611145, -0.521390438079834, -0.36997753381729126, -0.5325189232826233, 1.2281508445739746, 0.3703407049179077, -0.22426575422286987, 0.3306189775466919, 1.8667854070663452, -0.493432879447937, -0.15662530064582825, -0.11496659368276596, 0.9645814895629883, 0.24014735221862793, 0.4369887411594391, -0.16533023118972778, 0.4244564175605774, 0.5614476799964905, -0.8729563355445862, -0.006227841600775719, 1.746677279472351, -0.10749775171279907, 1.7037763595581055, -0.49592167139053345, -0.853839099407196, -0.26071980595588684, -1.4312069416046143, -0.3393156826496124, 1.1903876066207886, 0.7305769324302673], [0.7111939787864685, 0.07614380866289139, -0.1631489396095276, -1.2711373567581177, -1.7096482515335083, -0.9508009552955627, 0.26282912492752075, 0.012197992764413357, -0.026538683101534843, -1.3608722686767578, 1.5738447904586792, -1.8208534717559814, -0.08516889065504074, -2.9352550506591797, -0.7978156805038452, -1.9331674575805664, 0.38339921832084656, -0.4670446813106537, -1.1542245149612427, 0.5177279710769653, -0.7660685181617737, -0.7324713468551636, -0.9488295912742615, -0.5539054274559021, 1.624585747718811, 0.522257387638092, 1.5402588844299316, 1.4547770023345947, 0.812114417552948, -0.4446168541908264, -0.940485417842865, 1.062940001487732, -0.9031261205673218, -1.560259222984314, -0.7601631283760071, 0.6925244331359863, -0.22335846722126007, 1.0975478887557983, -0.3316269516944885, 0.4766092896461487, -1.3726047277450562, -1.5482800006866455, 0.16737394034862518, 1.15445876121521, -0.1358071267604828, 1.3694239854812622, 0.21384024620056152, 1.3818382024765015, 0.953110933303833, 0.7545493841171265, -0.1889282763004303, 1.6643694639205933, -1.4223906993865967, -0.5349518656730652, 0.15997059643268585, 0.284402072429657, -1.7667235136032104, -1.092879056930542, -0.9805040955543518, 0.7062609791755676, 0.2392130047082901, 0.7252724766731262, -0.7423120141029358, -1.336542010307312, 1.3239706754684448, -0.06756402552127838, -0.8920086622238159, 0.9365264177322388, 0.012876040302217007, 0.489056259393692, 0.46766331791877747, 0.012490482069551945, -1.3815265893936157, 1.9963176250457764, 0.6993412375450134, -1.5095967054367065, -1.1883430480957031, 0.744810163974762, -1.657027006149292, -0.7917162775993347, -0.9939121603965759, -0.959362268447876, -0.8480126857757568, 0.1627279371023178, 0.1585720032453537, 0.6698610186576843, -0.46400779485702515, 0.9180149435997009, -0.9179354906082153, -1.8205851316452026, 0.45082131028175354, -0.5880981087684631, -1.586516261100769, -1.5280861854553223, 0.8807098269462585, 1.7919448614120483, 1.5614160299301147, -0.9514268040657043, 1.5858092308044434, 0.14097540080547333, -0.20838238298892975, -0.12034260481595993, 1.4535640478134155, -1.4573333263397217, 2.3922548294067383, -1.5473383665084839, -0.5834100842475891, -0.3614451289176941, -0.23322223126888275, 1.3143383264541626, 0.8606141805648804, 1.6898155212402344, 1.31832754611969, 2.6290717124938965, 0.9344195127487183, -0.15982714295387268, -0.43363264203071594, 0.6573109030723572, -0.5888703465461731, -1.4435209035873413, 0.15409186482429504, -0.4261621832847595, 1.044999122619629, 0.6567505598068237, 0.32228079438209534, 0.8799705505371094, 0.058988504111766815, 0.5650275349617004], [-1.2130275964736938, -0.7960376143455505, 0.45034441351890564, -0.549869954586029, 1.6473604440689087, -0.9232761263847351, -0.12879124283790588, 0.4981548488140106, 0.7078630924224854, 0.23457299172878265, 1.4531584978103638, -0.37132528424263, -1.6341371536254883, 1.2339985370635986, 0.7763276696205139, 0.5421426892280579, -1.5516045093536377, -1.5676250457763672, -0.8447914123535156, 0.9536464214324951, -0.4926704168319702, 0.13897229731082916, 1.5009719133377075, -0.7868617177009583, 0.20316603779792786, -0.2116231918334961, -0.11411003023386002, -0.4594379663467407, -0.7223487496376038, 1.877820372581482, 0.30892878770828247, -2.4843649864196777, 1.0626615285873413, -0.8291293382644653, -0.9910612106323242, -0.02673276513814926, -0.7933704853057861, -1.4951977729797363, -1.4888702630996704, -0.000423264253186062, -1.2813822031021118, 0.6883595585823059, 0.058286942541599274, 0.6537775993347168, 0.786281168460846, -0.09200965613126755, -0.982719898223877, 0.36987173557281494, -1.079488754272461, -0.9583126306533813, -0.7499823570251465, -0.15775325894355774, 0.24252203106880188, -0.8132706880569458, -3.7730441093444824, -0.45877280831336975, -0.08690951019525528, -1.4421794414520264, -0.6104778051376343, -0.2340753823518753, 0.40943196415901184, -0.3880762457847595, -0.19457389414310455, 0.6275407671928406, 0.43134891986846924, -1.883959412574768, -0.3909747004508972, 0.5426942706108093, -1.5266193151474, -0.1745263636112213, 1.07944917678833, 0.5372724533081055, -0.6537395119667053, -0.37710586190223694, 0.1396797150373459, 1.8707796335220337, 0.23260045051574707, -0.3643359839916229, 1.7138310670852661, -0.5238508582115173, -0.9674363732337952, -2.11651873588562, 0.9732263088226318, 0.3298352062702179, -0.6911611557006836, 0.1490582823753357, -0.14456087350845337, -0.7605840563774109, -1.2502388954162598, 3.18911075592041, 0.1712087243795395, -0.5416553616523743, -1.0020196437835693, -1.5560405254364014, -0.16777877509593964, -0.13238359987735748, 0.30035650730133057, -0.02212369255721569, 1.433889389038086, 1.4188575744628906, -0.8770684599876404, 0.17195086181163788, -0.2116793841123581, -0.16554276645183563, 0.8407093286514282, -0.5936775803565979, -0.14873485267162323, 1.1889495849609375, -1.664837121963501, -0.2330765426158905, -0.6181179285049438, 1.7449541091918945, -1.2295827865600586, 1.5729596614837646, -0.1710413694381714, 0.8633202314376831, -0.10557961463928223, 0.08932037651538849, 0.3330734670162201, 0.8581804037094116, 0.797077476978302, -0.3092099726200104, 0.33875060081481934, -0.7401282787322998, 0.8947504758834839, -0.04918316751718521, -1.1237988471984863, 0.5323474407196045], [0.5416625142097473, -0.24922974407672882, -0.372859388589859, 1.7322096824645996, 0.28234952688217163, 2.154215097427368, -0.3637068569660187, 0.7473267912864685, -0.07955485582351685, 0.09997262805700302, -0.4218815267086029, 0.7116720676422119, 1.268849492073059, 1.2311607599258423, -0.7206701636314392, 0.9038335084915161, 0.8027087450027466, 0.45225733518600464, 0.6127714514732361, 0.3831543028354645, 0.20676888525485992, 0.11903472989797592, -0.5478797554969788, -0.404145210981369, -0.5134187936782837, -0.2345769703388214, -1.34102463722229, -1.2766810655593872, -0.5828160643577576, -0.8784600496292114, -0.11688201874494553, -2.1219899654388428, 0.23190587759017944, -0.18755009770393372, -1.1648058891296387, 0.15401042997837067, 1.0666371583938599, -1.1262511014938354, -2.776740074157715, -0.49642857909202576, -0.9126628041267395, -0.35277923941612244, -0.8789111375808716, 0.9535889029502869, 0.864729106426239, 0.8080670237541199, -0.12523002922534943, 3.200011730194092, -1.7393486499786377, -1.3407049179077148, 0.11233891546726227, 0.5567641854286194, 0.04729501157999039, -0.5343733429908752, 1.3090988397598267, -0.619529664516449, -0.787208616733551, 0.6838204264640808, 0.9765356183052063, 0.10677660256624222, 0.009250590577721596, 0.09886720031499863, 0.9485875368118286, -0.06232141703367233, -0.29264527559280396, 0.9163771867752075, -1.5323270559310913, -1.6104705333709717, 1.1146512031555176, -0.2403213381767273, 0.3089584708213806, -1.2729424238204956, 1.6288316249847412, 0.9729485511779785, -1.399179220199585, -1.405641794204712, -0.7583742141723633, -1.0964411497116089, 0.7305396795272827, -0.2543303072452545, -0.5553657412528992, 0.35924118757247925, -0.2059711068868637, 1.6802066564559937, 0.29370298981666565, -1.171485424041748, -1.3152275085449219, 1.4723399877548218, -0.0006655167671851814, -1.4728748798370361, 0.048826880753040314, -1.941772222518921, -0.7116553783416748, 1.4311233758926392, 0.20386911928653717, -1.2598130702972412, 0.11492238193750381, -1.1548757553100586, -0.04920065030455589, 0.038254544138908386, 0.6550358533859253, 0.8857755661010742, -1.680516242980957, -0.2234760820865631, 0.7698577046394348, 0.0061018499545753, -0.3947787880897522, 0.9215389490127563, 1.0447345972061157, -0.17372113466262817, -0.1504661589860916, 1.9136204719543457, -1.4614622592926025, 0.9542070031166077, 0.8796533346176147, 0.9600415825843811, -0.4083768427371979, 0.0202261283993721, -0.7441156506538391, 2.105384349822998, -0.8965306878089905, 1.8663647174835205, 0.41779568791389465, -0.9471356272697449, 0.1084754690527916, -0.6561342477798462, -0.07620783895254135, 1.1609898805618286], [-0.2218446433544159, 0.41288331151008606, 0.613877534866333, 0.9419043064117432, -0.6675015687942505, -1.4533164501190186, 0.014911382459104061, -1.7325996160507202, -0.9407525062561035, 0.993804931640625, -1.5708168745040894, -0.5328064560890198, 0.10546454042196274, 2.0816116333007812, 0.2290005385875702, 1.6053898334503174, -0.571563720703125, -0.6839119791984558, 0.2729320526123047, -0.6598882675170898, 1.5264310836791992, -1.4165316820144653, 0.43862438201904297, -1.2476710081100464, -0.9300981760025024, -1.3892912864685059, -1.856392741203308, 0.25508397817611694, 0.4731276333332062, 1.9646291732788086, -0.39282020926475525, -1.3115969896316528, 0.5435109734535217, 1.0376694202423096, -0.8880549073219299, -0.40347835421562195, 0.998775064945221, 0.06729438900947571, 0.034681662917137146, 0.5146268606185913, 0.14755871891975403, 0.8917171955108643, -0.7587729692459106, 2.3110382556915283, 1.2972217798233032, -1.9756183624267578, 0.46113520860671997, 0.2741577625274658, 0.5956459045410156, 0.7718747854232788, 1.1711524724960327, 0.1460457146167755, 1.8022980690002441, 0.032598573714494705, -1.4616063833236694, 2.0247745513916016, -0.5499706268310547, -0.7909201383590698, -0.36847877502441406, 0.07467032968997955, -0.8133097290992737, -0.23369263112545013, -0.7550239562988281, -0.8509591817855835, 1.9443732500076294, 1.7051324844360352, -2.0072572231292725, -0.29013362526893616, 0.4822498857975006, 1.1803064346313477, -0.6988587379455566, -0.4940236210823059, -0.09207472205162048, 0.2065260261297226, 2.95109486579895, -0.17992758750915527, 1.079593539237976, 1.486094355583191, 0.6078359484672546, 1.0755071640014648, -0.16525860130786896, -1.3239837884902954, -0.0511368028819561, -2.0862390995025635, 0.16967539489269257, -0.027758438140153885, -0.047093238681554794, -0.5360138416290283, -0.013598345220088959, 1.2596591711044312, 1.7769793272018433, -1.979022741317749, 1.9412477016448975, 0.8421977758407593, 0.23346711695194244, 0.6733512878417969, 0.8052383065223694, 1.6767271757125854, 0.6455028057098389, -0.18804562091827393, 0.8739592432975769, -0.21662305295467377, 0.37251612544059753, -0.13263945281505585, -1.445876121520996, -0.7942376136779785, -0.9617258906364441, 0.9220829010009766, -2.6109957695007324, 0.20382380485534668, -0.5286688804626465, -2.419308662414551, 1.0862029790878296, -0.6256822943687439, -0.8049728274345398, -0.9731770157814026, -0.9749762415885925, -0.6335546970367432, 0.07054990530014038, -0.21317580342292786, 0.14986790716648102, -0.7659560441970825, -0.9510533213615417, 0.775685727596283, 0.2074446827173233, -0.29372090101242065, -0.06281336396932602, -0.12761054933071136], [1.6670403480529785, -0.1768956184387207, 1.725670337677002, -1.5167564153671265, -0.44003134965896606, 0.048213351517915726, -0.6707023978233337, 0.32958462834358215, -1.7366615533828735, 0.978859543800354, 0.050808511674404144, -1.9482760429382324, 0.1281833052635193, 0.42230331897735596, 2.1908271312713623, -0.18618904054164886, -0.69855797290802, -1.1599472761154175, 0.06295858323574066, -1.1146694421768188, -1.1023796796798706, 0.9049901962280273, 0.005969286896288395, 0.10861591249704361, -0.05701328441500664, 0.9232515096664429, 1.339913249015808, 0.8346011638641357, 0.3298114240169525, 0.15335795283317566, -0.747366726398468, 0.7464591860771179, -0.36792197823524475, 0.10693079233169556, -0.051490604877471924, 0.01144234649837017, -1.4149525165557861, -0.5759518146514893, 0.9410862922668457, 0.7344083786010742, 0.1981431543827057, -0.12697559595108032, 0.6798157095909119, 0.4340590834617615, -0.6904511451721191, -2.9098052978515625, 0.12912185490131378, 1.6940351724624634, -1.9078491926193237, 0.9383373856544495, 1.7028777599334717, 1.4697457551956177, -0.5559369921684265, 0.3359204828739166, -0.01316993497312069, 0.5677217841148376, -1.3028937578201294, 1.644720435142517, 1.5100982189178467, 1.07806396484375, -0.2373928725719452, 0.29384806752204895, -0.9278531670570374, 0.33614060282707214, 0.9654854536056519, -0.08146446943283081, -0.2596766948699951, -0.07739735394716263, -1.915992259979248, 1.52409029006958, -0.04335501790046692, 0.035216886550188065, -1.4507156610488892, 2.423572540283203, -0.4968639016151428, 1.0835014581680298, 0.6099899411201477, 0.5440248847007751, 0.584617555141449, 1.410194993019104, 0.14619670808315277, 0.5245010852813721, -0.301608145236969, 0.2321649193763733, 0.3479405343532562, 1.3169822692871094, 1.438963770866394, 0.7296742796897888, 0.4194190800189972, 1.0129005908966064, -0.4138842523097992, -0.9423294067382812, -0.13499420881271362, -1.4214485883712769, -0.02772088348865509, 0.501078724861145, 2.1032016277313232, 0.9345081448554993, 0.564211905002594, -0.3252430856227875, -0.9921952486038208, 0.4311278760433197, 0.12278297543525696, 1.6900031566619873, 0.9373790621757507, -2.922879695892334, -1.4209967851638794, 0.7438976764678955, 0.681761622428894, 1.8680760860443115, -0.25921714305877686, -1.5897105932235718, 0.5444952249526978, -0.44105273485183716, -0.18934577703475952, -0.6434972286224365, 0.36697426438331604, 0.032747358083724976, -0.4047519266605377, 0.31306925415992737, -0.2839014232158661, 1.0186847448349, -0.19765715301036835, -1.4472757577896118, -0.23853087425231934, 0.6496047377586365, -1.0224577188491821, -0.35543742775917053], [-0.006817151326686144, 0.8706572651863098, -0.37177684903144836, -0.3544550836086273, 0.792864203453064, -0.9202584624290466, -0.29332906007766724, 0.8406191468238831, -0.05168132111430168, -0.7524322271347046, 1.4684672355651855, 0.26414141058921814, 0.9266420006752014, 0.04698558524250984, -1.3217968940734863, -1.2830545902252197, 0.9869229793548584, -0.18225981295108795, -0.6126751899719238, -2.105632781982422, -0.9078271985054016, -0.31206071376800537, 1.1348577737808228, 1.0888011455535889, 1.0833309888839722, 0.0019621376413851976, -0.9695906639099121, 0.07706847786903381, -0.8954877853393555, -1.1881256103515625, 1.7096972465515137, -0.36548393964767456, -0.24122220277786255, -0.03241785988211632, 0.09404121339321136, -1.114895224571228, 0.04039841517806053, -0.3560720682144165, 0.19190646708011627, -0.11561308056116104, -1.3943705558776855, 0.10216250270605087, 1.4647542238235474, -0.2774444818496704, -1.072119951248169, 0.010656925849616528, -1.8704509735107422, 0.34640705585479736, 0.0572049543261528, 0.4525834321975708, 0.9531434178352356, 0.17841875553131104, 0.660263180732727, -0.6664554476737976, -1.1577188968658447, -0.8569221496582031, 1.7951469421386719, -2.0175888538360596, 0.20597706735134125, 1.1802263259887695, -0.038786381483078, -1.3295342922210693, -1.1805636882781982, 0.7449964880943298, 0.22648969292640686, 0.6030793786048889, -0.6097357273101807, -1.2404578924179077, 2.267033338546753, -1.8447386026382446, 0.6275142431259155, -0.6921796798706055, -0.8371530175209045, 0.18663708865642548, -2.329401969909668, 0.5419010519981384, -0.26112625002861023, 1.248471736907959, -1.1578006744384766, -0.3708428740501404, 0.6032923460006714, -0.18338394165039062, 1.2167668342590332, 0.15213890373706818, -2.74068546295166, 1.5562660694122314, 0.2734597623348236, -0.5888414978981018, 0.03663342073559761, -0.9418658018112183, -0.12384118884801865, -0.9132907390594482, 0.31971192359924316, -0.34996357560157776, 2.2044177055358887, -1.5786855220794678, -0.9549612402915955, 0.6483140587806702, -0.2188820242881775, -1.3388994932174683, -1.6280394792556763, 1.2600419521331787, -0.28681209683418274, -1.035074234008789, 0.033790722489356995, 0.6372655034065247, -1.8257114887237549, -0.6680780649185181, -0.5618223547935486, 1.48576021194458, -0.3393840193748474, 0.7229838371276855, 0.3199632465839386, 0.16483183205127716, -0.5481902360916138, 0.10160905122756958, 1.7844452857971191, -0.6685019731521606, -0.12115449458360672, 0.5087855458259583, -0.4244735538959503, -1.4698344469070435, -1.7839618921279907, -0.370900422334671, 1.7429580688476562, -0.32227301597595215, 0.855880856513977, -0.8285366296768188]], [[-0.5725002884864807, 0.6395916938781738, 0.15249383449554443, -0.2652405798435211, 0.5688409209251404, -0.5046730637550354, 1.5183918476104736, -0.15063300728797913, 0.5235198736190796, -1.0985617637634277, -0.6840851902961731, -0.19839751720428467, 0.020620597526431084, -0.9430146813392639, 0.6383148431777954, 0.2650532126426697, 1.5296709537506104, 0.1435292661190033, 1.1976981163024902, -1.622244954109192, -0.2018374651670456, -1.7942590713500977, 0.15418177843093872, 0.3158662021160126, -1.9610038995742798, 0.5733601450920105, -0.8911686539649963, 0.2731156051158905, -0.10539348423480988, 0.5534077286720276, -0.3256268799304962, 0.3604715168476105, 1.5341215133666992, 0.08529774099588394, 0.04456669092178345, 1.4496219158172607, 1.5537924766540527, 0.6789305806159973, 0.3509331941604614, 0.021555280312895775, -0.5258382558822632, -0.1718006283044815, -0.5207865238189697, 0.2662879526615143, 2.0553812980651855, 1.3361196517944336, 1.0930200815200806, -0.18911445140838623, -0.9257494211196899, -1.6579930782318115, 0.7885884046554565, 0.17095668613910675, 1.4963529109954834, -0.017588350921869278, -0.42764005064964294, -0.2020416557788849, -0.7439334392547607, -0.46844175457954407, -0.34892308712005615, 0.1420673131942749, 0.9648298621177673, 1.1185334920883179, 1.7594377994537354, -1.3533363342285156, 0.5217671990394592, -1.4386365413665771, -0.7225812077522278, -0.44797831773757935, -1.8408247232437134, -0.10301349312067032, 2.224648952484131, -1.945236086845398, -0.7608911991119385, 1.0962374210357666, -0.017515525221824646, -0.42652449011802673, -1.5620092153549194, 0.1670488715171814, 1.1556687355041504, 1.1995594501495361, 0.40997931361198425, 0.1096080020070076, 0.02481924556195736, -0.23215018212795258, 0.017426177859306335, -0.7871604561805725, -0.6921052932739258, -1.2155364751815796, -0.02292201854288578, -0.1793200969696045, -0.4541158378124237, 1.8522968292236328, -0.45183065533638, -0.5290345549583435, 0.9884675741195679, 0.40675443410873413, 0.5142796039581299, 0.5905113220214844, 0.43933263421058655, -1.8784137964248657, 0.17790858447551727, 0.8469825983047485, -1.0026190280914307, 1.7375472784042358, -0.16251946985721588, -0.6198983788490295, -0.6669979691505432, -1.1246713399887085, -0.31128108501434326, 0.05827844515442848, 0.2611726224422455, -0.5125175714492798, -0.14873464405536652, 1.4195307493209839, -1.938218116760254, 2.092294931411743, -2.040945529937744, 1.7400729656219482, -1.037152886390686, 0.14813832938671112, 0.6402760148048401, 0.06326203048229218, -0.016748854890465736, 0.2552996873855591, -0.9483239054679871, -1.131272792816162, -1.268911361694336, 0.4154469072818756], [0.5989309549331665, -1.3577195405960083, 1.8037089109420776, -1.8154939413070679, -0.5654248595237732, -0.8218743205070496, -0.5792039632797241, -0.9664804339408875, -1.608961582183838, -0.3450433313846588, -0.4417625367641449, 1.5557447671890259, 0.647538423538208, -0.8657087683677673, 0.7043038606643677, 0.9842328429222107, 1.173046350479126, 0.32677435874938965, 1.139050006866455, 1.2549176216125488, 0.1687604784965515, -1.1029809713363647, -0.9734249711036682, -0.41223910450935364, 1.1328940391540527, -1.2773005962371826, -1.233566403388977, -1.9253394603729248, -0.18566375970840454, -1.0292493104934692, -1.1900185346603394, 2.0413429737091064, 0.39476823806762695, -0.22881364822387695, -0.3987491726875305, -0.22706468403339386, 0.9133926033973694, 1.3552132844924927, -1.0289359092712402, 0.9521804451942444, -0.5953438878059387, -0.7294743657112122, -0.41448095440864563, 1.1503617763519287, -0.14707733690738678, 1.0795536041259766, 0.6698659658432007, 0.9020075798034668, 0.7552838325500488, -1.5707924365997314, -0.4151710569858551, -0.4087394177913666, -1.3533207178115845, -0.4162290692329407, 1.1335686445236206, -0.013859988190233707, -1.7692346572875977, -0.7822070121765137, 0.7143632769584656, -1.096058964729309, -0.49491435289382935, 0.10041302442550659, -1.4172247648239136, 1.8269141912460327, -2.487338066101074, -0.2882460355758667, -0.0616760216653347, 0.9137367010116577, 0.8072184920310974, -2.0488948822021484, -0.3824201226234436, 0.8617417812347412, 0.4777831435203552, -0.9404864311218262, 0.5212022662162781, -1.4098007678985596, 0.2977103888988495, 0.05323812738060951, -0.17830753326416016, -1.188788652420044, -0.12000267207622528, -0.7117195725440979, -1.0410884618759155, -0.010477602481842041, 1.0891227722167969, -0.3465997874736786, 0.29176196455955505, -2.1926727294921875, 0.14800795912742615, 0.1712264120578766, 2.2608346939086914, -0.8364176154136658, -0.30967432260513306, -0.13957448303699493, 0.22881609201431274, -1.763411045074463, -1.1690993309020996, -0.7633727788925171, -1.0196771621704102, 0.1950637698173523, 0.2848065495491028, -2.1681008338928223, -1.328937292098999, 0.2175351083278656, -0.4090108871459961, -2.2137725353240967, 1.3087399005889893, 0.7493249773979187, -0.11088462173938751, -1.3200002908706665, 0.048159677535295486, 1.2671297788619995, 0.33623766899108887, 0.5133148431777954, 0.47000062465667725, -0.6166223287582397, 0.34347304701805115, 0.325053334236145, -1.6384514570236206, -0.31315070390701294, -0.13687697052955627, 1.1315621137619019, 1.1801259517669678, 0.755716860294342, -1.1660704612731934, 0.47198617458343506, -0.5446887016296387, 1.0368598699569702], [-0.23770339787006378, -0.643231213092804, -1.4298386573791504, 1.6053677797317505, 0.6570138335227966, -0.2081068754196167, -1.2248281240463257, -0.5361890196800232, -0.7097491025924683, 1.385135293006897, 1.542972207069397, 1.1995905637741089, -0.9319091439247131, 0.07638399302959442, 0.40299659967422485, 1.2625786066055298, 0.22552300989627838, 1.3079867362976074, 0.5710467100143433, -1.1771965026855469, -0.8054138422012329, -0.1909012645483017, 0.25783923268318176, -0.9101435542106628, -0.3782863914966583, -0.26183804869651794, -1.2425787448883057, -0.9438900351524353, -0.07931728661060333, -1.0004605054855347, -0.009510545991361141, -0.10423479229211807, 0.7215794920921326, -0.6794062256813049, 1.9392024278640747, 0.5795093774795532, 0.550150990486145, -0.011649495922029018, -0.03777800127863884, -1.2396408319473267, 0.4917617738246918, -1.6350935697555542, -0.8164727687835693, 0.6072922945022583, 0.046599380671978, -0.9616785645484924, -0.7688961029052734, -0.5921632647514343, -0.27128544449806213, 0.223343163728714, -2.3142130374908447, 0.17473134398460388, 0.5268197655677795, 0.6315581798553467, -0.50040602684021, -0.2959236204624176, -0.38235071301460266, -0.2429480403661728, 0.3520110249519348, -0.9914562702178955, 1.6964970827102661, 0.2454051822423935, -0.6673400402069092, 0.6397612690925598, -1.3822702169418335, -0.9493578672409058, -0.945156455039978, 1.5671875476837158, 0.7230704426765442, 0.04990440979599953, 0.05228878930211067, 1.2463841438293457, -0.8383467197418213, 0.7788567543029785, 0.4870634078979492, -0.7779896855354309, 0.5698527693748474, 1.920518398284912, 0.25940924882888794, 0.5963723659515381, -0.45295560359954834, 0.6071323156356812, -0.4942294955253601, 0.6711921691894531, 0.5953647494316101, 0.10648172348737717, -1.592737078666687, -1.9441347122192383, -0.4840203523635864, -0.46989625692367554, -1.4412450790405273, -1.6082775592803955, -0.2422305941581726, -0.04315375164151192, -1.3310470581054688, 0.6464231610298157, 0.8053038120269775, 1.125725269317627, -2.3031928539276123, -0.04797890782356262, -0.1994129717350006, -0.2715624272823334, -0.0541488379240036, -0.13354803621768951, -0.49518316984176636, 0.6765474081039429, -0.8417556285858154, -1.4168225526809692, 0.7801311612129211, 0.1731923520565033, 1.3174277544021606, -0.29351601004600525, -2.270324945449829, -0.7526471614837646, -3.170257806777954, 0.7945679426193237, -0.5590518116950989, 1.5158859491348267, 0.24141700565814972, -0.27387112379074097, 0.43080365657806396, -1.7186216115951538, 2.577038526535034, 0.7375999689102173, 1.2273677587509155, -1.5048394203186035, 0.16992467641830444, -0.7552562355995178], [-1.7833483219146729, 0.588412344455719, -0.805276095867157, -1.6360825300216675, 0.6748558282852173, -0.7577725648880005, -0.049952346831560135, 0.2254381626844406, -1.5612118244171143, 0.4091368615627289, 0.9405937790870667, 0.6080530285835266, 2.160717010498047, 1.2995851039886475, -1.1407668590545654, 0.5503497123718262, 0.6603304147720337, 0.2026432901620865, -0.19325189292430878, 0.3291890621185303, 1.340923547744751, -0.5616082549095154, -0.9199522733688354, 1.5866233110427856, -0.9270941019058228, -0.04558328539133072, 0.35765689611434937, -0.6324208974838257, 0.5452196598052979, -0.277210533618927, 0.2920595109462738, -0.5656442046165466, 0.27297812700271606, -0.48771795630455017, -0.4287415146827698, -0.48511025309562683, -0.5593127012252808, 0.7278178930282593, 1.0973867177963257, -1.4440770149230957, 0.03771572560071945, -0.11633948981761932, -0.14733107388019562, 0.09456289559602737, 0.561762809753418, 0.2891281843185425, -0.4678719937801361, -0.011555829085409641, -0.3895258903503418, -0.6288477182388306, 0.28262534737586975, 0.6508409380912781, -0.08316915482282639, 0.41968950629234314, -0.627357542514801, 0.8793299198150635, 0.21092283725738525, 0.684941291809082, 1.0673109292984009, 0.463379442691803, -0.12757675349712372, -2.2391791343688965, -1.0677355527877808, 0.34809306263923645, -1.0806456804275513, 1.1464166641235352, 0.8031231164932251, -0.26108962297439575, -0.5187751054763794, -0.3822818696498871, 0.29342004656791687, -1.533624529838562, 0.3343389630317688, 1.5275582075119019, 0.9952265024185181, 1.4568129777908325, 0.01568557694554329, -1.8661425113677979, 0.6553802490234375, 0.1137896403670311, -0.6478800177574158, -0.4586535096168518, -0.8793109655380249, 0.2949727475643158, 0.13758252561092377, 1.2876564264297485, -0.3012067973613739, -1.252175211906433, 0.6182869076728821, -1.2711917161941528, 0.4647214114665985, 0.8558141589164734, -1.6634314060211182, -1.1312503814697266, -0.7758410573005676, 2.197399377822876, 1.083418846130371, 2.0300159454345703, 0.5102919936180115, -0.42684367299079895, 0.5056633949279785, -1.1273581981658936, 1.8864696025848389, -0.27851155400276184, -0.08054209500551224, -1.238196849822998, 1.2193180322647095, -0.40863171219825745, 0.2717653512954712, 0.6074869632720947, 0.753424346446991, -0.309255987405777, 0.08022075891494751, -0.4280596077442169, -0.8552585244178772, 1.075244426727295, -1.8532278537750244, -0.31702035665512085, 0.5728091597557068, 1.0039490461349487, -0.5542454123497009, -1.141619324684143, -1.6486854553222656, 1.6470164060592651, 1.1003168821334839, 0.38644716143608093, 0.6387050747871399, 2.3714070320129395], [0.3997126817703247, 1.0216898918151855, -0.3496631681919098, 0.03092336468398571, 0.6202355623245239, -0.41671597957611084, -0.5178791880607605, -0.3398224413394928, 1.5092397928237915, -0.9397544860839844, 0.5513192415237427, -0.008606499992311, 0.6873704195022583, -0.6541728973388672, 0.2884947955608368, 1.176797866821289, -0.5903239250183105, 1.3094627857208252, 1.1978932619094849, -1.8891633749008179, 0.5218955874443054, -0.033771026879549026, -1.3965113162994385, 1.0391547679901123, -0.985081136226654, 1.070554256439209, 0.5856937766075134, 0.8853113055229187, -0.6753894686698914, 0.36774542927742004, 0.030164405703544617, -0.7275821566581726, 0.01722363568842411, 1.3328830003738403, 0.242047980427742, -1.603463053703308, -1.1192400455474854, -0.569337785243988, -0.8316349387168884, 0.8485398292541504, -1.4407724142074585, 0.235457643866539, 1.4067795276641846, -0.891208291053772, 0.2996666431427002, -1.5836604833602905, -0.3367127776145935, 0.291687548160553, 1.8368782997131348, 1.4948012828826904, 0.6690189838409424, -0.506860613822937, -1.0890510082244873, 0.5826745629310608, -0.46905970573425293, -0.3278440237045288, -0.8612530827522278, -3.195528984069824, -1.6744948625564575, -3.2806131839752197, 1.6199408769607544, -0.5542622804641724, -1.5416547060012817, 1.6238678693771362, -0.8825649619102478, -0.08092091232538223, 0.9982504844665527, -0.16708897054195404, 1.7278430461883545, -0.13571009039878845, -2.1437644958496094, -0.3101878762245178, -0.34616950154304504, 1.6849263906478882, -0.2706463038921356, 1.235730767250061, -0.24861806631088257, -0.37410590052604675, 0.060395728796720505, 0.41343098878860474, 0.5549463033676147, 0.8895481824874878, -0.5122288465499878, 0.08172985911369324, -0.003079708432778716, -2.971346139907837, 1.2676324844360352, -1.518789291381836, -1.1351921558380127, -0.620041012763977, 0.5506871342658997, -0.8953604102134705, -0.20741967856884003, 2.6894376277923584, -0.05842306464910507, -0.035526663064956665, 0.44499921798706055, 1.4906154870986938, 1.5652142763137817, 0.5879684090614319, 0.24016381800174713, -1.0620083808898926, -0.13721877336502075, 0.43684011697769165, 0.20749512314796448, -0.4035069942474365, 0.9642515778541565, 0.5949954390525818, -0.7356957197189331, -1.1108434200286865, 0.47147509455680847, -0.5132167935371399, -0.3590150773525238, 0.669925332069397, 0.5342057347297668, 0.9123018383979797, 0.997934877872467, 0.7177188992500305, 0.052537936717271805, -0.22711633145809174, 2.8739688396453857, 2.2651708126068115, 0.449700266122818, 0.7112902402877808, -0.3230159282684326, -0.019224083051085472, -0.7076782584190369, -0.5788192749023438], [0.6851774454116821, 1.1331902742385864, 2.299671173095703, -1.050024151802063, 0.8028264045715332, -0.9199905395507812, -0.22096772491931915, 0.9934349656105042, 0.993185818195343, 0.9160696864128113, -0.5051433444023132, -0.966256856918335, -1.0549606084823608, 0.022296594455838203, -1.0376487970352173, 0.7361504435539246, 1.3205958604812622, 0.6358805298805237, -1.1687325239181519, -1.150564432144165, -0.27895453572273254, 0.7768778800964355, 0.19975417852401733, 0.5436021089553833, -1.3935801982879639, -1.176105260848999, 0.47428154945373535, -0.5084579586982727, -1.080068588256836, -2.07918381690979, 1.0836336612701416, 0.06681285798549652, -1.0917388200759888, -0.5032937526702881, -0.2852800190448761, -0.19998309016227722, 1.655198097229004, 1.343253254890442, -0.07799271494150162, -1.3510288000106812, -0.49376824498176575, -0.5155573487281799, -0.7590166330337524, -0.8353922963142395, -1.7753831148147583, 0.04828878119587898, -0.4401291310787201, -0.29940614104270935, -0.22299788892269135, 0.04918954148888588, 0.9213495254516602, 1.1078836917877197, 0.8213827013969421, -1.5614573955535889, 0.394873708486557, 0.16145117580890656, 0.16818013787269592, -1.122623085975647, -0.4724062383174896, 1.4887540340423584, -1.0766382217407227, -0.7541698813438416, 1.0003173351287842, 0.5168557167053223, -0.11273466795682907, -0.5495786070823669, 0.45633548498153687, 0.26536425948143005, -0.25684091448783875, 0.2362929880619049, -0.7247530221939087, 0.8264597654342651, -0.11898829787969589, -0.11679007112979889, -0.06124985218048096, -1.2094804048538208, -0.18218378722667694, -1.726483941078186, -0.1370922178030014, 0.3126285970211029, -1.444705605506897, -1.368755578994751, 1.7874339818954468, 0.26462703943252563, -2.1508615016937256, 1.5026593208312988, 0.8696133494377136, -0.44734033942222595, 0.11470939218997955, -0.00968255940824747, -0.8926493525505066, 0.42432984709739685, 0.04819145053625107, -0.8159494996070862, 0.42826053500175476, 0.788192868232727, 1.4633439779281616, 0.29418566823005676, -1.4859752655029297, -0.4016268849372864, 0.8696155548095703, 1.2485500574111938, 0.29354017972946167, -0.8970584273338318, -0.6161915063858032, -0.7215410470962524, 0.3351750075817108, 0.8209142088890076, -0.07747063040733337, 0.637588620185852, 1.9064438343048096, -0.47456926107406616, -0.02068149857223034, -1.4621702432632446, -0.5171434283256531, -0.37313535809516907, -1.3923683166503906, 1.43264901638031, -2.1321394443511963, -1.0671647787094116, 2.172421455383301, 0.7898533940315247, -0.2157326191663742, -0.4025631248950958, -0.8610366582870483, -0.19041746854782104, 2.244127035140991, 1.2771122455596924], [0.3250466287136078, -0.15259414911270142, 0.5871431827545166, 1.1099181175231934, 0.34033703804016113, -1.3208342790603638, 1.2468417882919312, 0.3472430109977722, -0.39159998297691345, 0.5738029479980469, 0.2276063859462738, -1.3562564849853516, 0.2873029410839081, -1.7221457958221436, 1.0059146881103516, -0.4561605453491211, -0.7946188449859619, 0.5443730354309082, 0.7199884057044983, 1.3358385562896729, -0.08271622657775879, -0.7817784547805786, 1.3103126287460327, -0.6146937012672424, -1.3817873001098633, -0.19000211358070374, 0.7905876636505127, -1.4566478729248047, 0.6072104573249817, 1.7417420148849487, -1.3152844905853271, 0.1441165804862976, 0.08295997977256775, -0.30570217967033386, -0.8660022616386414, 2.603883981704712, 1.1669130325317383, -1.4782768487930298, 0.8940297961235046, -0.5980209708213806, -1.2864338159561157, 0.2032444030046463, -0.1337127983570099, 0.13323357701301575, -1.7238472700119019, 0.8124263882637024, -1.0591822862625122, -1.1962796449661255, 2.5354363918304443, -0.8170068264007568, -0.741606593132019, -1.2504661083221436, 1.8041579723358154, 0.3909158408641815, -1.3044897317886353, -1.344367504119873, -0.28779637813568115, 0.6947163343429565, -2.8000917434692383, -1.00750732421875, -0.4910967946052551, -1.357934594154358, 0.8119722008705139, 1.962623119354248, -0.49544504284858704, 0.5871456861495972, 0.05156215280294418, -0.8339313268661499, 0.3329036235809326, 0.3007547855377197, -1.1922310590744019, -1.1711902618408203, 0.003663767594844103, -1.1943966150283813, 0.991912305355072, -1.013676643371582, -0.27481117844581604, 0.07176998257637024, 0.026305917650461197, -0.5628872513771057, 0.6764388084411621, 0.8176336884498596, 0.3730364441871643, -0.4107167422771454, -1.395161747932434, 1.4852937459945679, -0.0191049724817276, -1.9540808200836182, -1.0858235359191895, 1.4440324306488037, 0.9618337154388428, -0.2909720242023468, 0.5958716869354248, -0.7332944273948669, -1.684049367904663, 1.9421051740646362, 0.05776989832520485, -0.18756352365016937, 0.07893785834312439, -0.22843149304389954, 0.3282323479652405, -0.639202892780304, -0.5862147808074951, 1.314744472503662, 0.15959994494915009, 0.8176035284996033, 0.26171737909317017, 0.6261021494865417, -0.9428250193595886, 1.5910698175430298, 0.7568657994270325, -1.2843539714813232, 0.8466291427612305, 0.7819045186042786, -0.12292401492595673, 0.9773765802383423, -0.1934974044561386, 1.065443754196167, -0.3681338131427765, -0.4912339746952057, -0.918576180934906, -0.07411391288042068, 2.970055341720581, -1.0436301231384277, -2.7924721240997314, -0.18449433147907257, -0.0037414999678730965, -0.06592631340026855], [-0.2932111620903015, -0.13230310380458832, -1.2193635702133179, -1.3454405069351196, -1.2257890701293945, 2.3709757328033447, 0.3356644809246063, -1.189429759979248, -0.29584798216819763, 1.8873223066329956, 0.9437187314033508, 0.28270259499549866, -0.5280676484107971, -0.8859721422195435, 0.4894748032093048, 0.4059118330478668, 0.6437531113624573, -0.6581332087516785, -0.1565907746553421, -1.5926812887191772, -0.15570203959941864, -0.48101699352264404, -0.16970543563365936, 1.6105509996414185, 0.7900223731994629, -1.9880656003952026, 0.6306731104850769, 0.7230502367019653, 1.3087550401687622, -0.14701135456562042, -0.3809206783771515, 0.06876198202371597, -1.3908506631851196, 1.2631219625473022, -0.1414335072040558, -0.9859979748725891, -0.4696592688560486, -0.360839307308197, 0.42708826065063477, -0.9623737931251526, 0.3121051490306854, -0.751729428768158, 0.2196788638830185, -0.5943358540534973, -0.27446699142456055, -0.23504415154457092, 0.28657445311546326, -0.07134414464235306, -0.19723182916641235, 0.2601638436317444, -2.6315996646881104, 0.7129616141319275, -1.9914530515670776, 0.43326708674430847, 1.135970950126648, 1.1711643934249878, 0.43904611468315125, -0.069748654961586, 0.4933331608772278, -0.04286406934261322, 0.5191757678985596, 0.555979311466217, 0.9454874992370605, 0.47990354895591736, -0.03895368054509163, 1.3601902723312378, 1.8080729246139526, -0.9074551463127136, -0.5983127355575562, 1.8859246969223022, 0.1746116280555725, -2.1850719451904297, 0.7992674112319946, -0.8717492818832397, 0.9890255331993103, -0.9735199213027954, 0.4396806061267853, 0.1381695419549942, -1.7027114629745483, 1.4151607751846313, -0.22204124927520752, -0.8125633001327515, -0.20315414667129517, 1.651382327079773, 0.7569078803062439, -0.5839707851409912, 0.7900010943412781, -0.30880194902420044, -0.15175698697566986, 1.8549470901489258, -0.7838394641876221, 0.5533854365348816, -1.194663643836975, 1.734683871269226, -0.9006947875022888, 0.8986705541610718, 0.8383654952049255, 1.5239152908325195, -0.3260570466518402, 1.8827604055404663, -0.3018888533115387, 0.08275879919528961, 0.6578809022903442, -0.3511315584182739, 0.6958437561988831, -1.967590093612671, -1.32901132106781, 1.0819860696792603, 1.9034242630004883, -1.1149834394454956, 0.33824795484542847, 0.8333758115768433, -0.009518750943243504, -1.6837308406829834, 0.512783944606781, 0.4117920696735382, -0.2815984785556793, 0.8109340071678162, -0.22155900299549103, -0.7319427132606506, -0.3733387887477875, 0.5084749460220337, -0.31791478395462036, -1.28507661819458, 0.04410526156425476, -0.7309058308601379, 2.583252429962158, -0.36524802446365356]], [[0.4458027184009552, -1.0283819437026978, 1.135585069656372, -0.10378863662481308, -0.8363596796989441, 0.34711411595344543, 0.5954790115356445, 1.007624864578247, -0.35000303387641907, -0.19299082458019257, 1.9126474857330322, -1.2812443971633911, -2.0306379795074463, -0.03802497684955597, 0.049854688346385956, 1.348982334136963, 2.6276683807373047, 0.6380600333213806, 0.08526083081960678, -0.6532680988311768, -0.3322790265083313, 0.5639346837997437, -0.9402918219566345, -1.4031275510787964, 0.9574312567710876, 0.40403833985328674, -1.3041274547576904, -1.1852004528045654, 0.14350053668022156, -1.1725398302078247, 0.37369269132614136, 0.04317717254161835, 0.7401252388954163, 1.4231541156768799, 1.6200065612792969, 0.9974777102470398, 0.39548665285110474, -0.7290312647819519, 0.5020437836647034, -1.9650120735168457, -0.14331339299678802, -0.39975178241729736, 0.7888991236686707, -2.027935028076172, -0.7100117206573486, 0.2517658472061157, -1.3376213312149048, -2.3380424976348877, 0.4363122880458832, -0.15387941896915436, -0.7330299019813538, 0.5625481009483337, -1.3041110038757324, 0.9124846458435059, -1.667775273323059, -0.3374188542366028, -0.23199811577796936, -1.6722126007080078, -0.37631499767303467, -1.7202951908111572, 1.7394657135009766, -1.5639516115188599, -0.3590957522392273, -0.7182158827781677, 0.13934507966041565, -0.12294085323810577, -0.8363928198814392, -0.3231372535228729, 0.9963061809539795, 0.989774227142334, -0.5753482580184937, -0.28158125281333923, 1.8369290828704834, -1.1293270587921143, 1.5309429168701172, -0.43548259139060974, 0.14729110896587372, 0.012357402592897415, -0.4234515130519867, 0.023043598979711533, 0.3511507511138916, 1.0434422492980957, -0.8693686127662659, 0.9757078886032104, 1.7026041746139526, 0.8060442805290222, 0.6893900632858276, -0.11777328699827194, 0.13101354241371155, -0.6161162257194519, 1.1008920669555664, -1.32524573802948, -2.7006962299346924, -0.7517339587211609, -0.6956816911697388, 0.33687636256217957, 0.9340855479240417, -0.998024582862854, -0.19047388434410095, -0.430974543094635, -0.9606636762619019, 1.255135416984558, 0.10969240963459015, -1.0454825162887573, 0.4102937877178192, 1.3340444564819336, -0.3267747163772583, 0.9602270126342773, -0.04541527107357979, -1.0728493928909302, -1.7060884237289429, 1.217468023300171, 2.110417127609253, -0.05146408453583717, 1.0840917825698853, -0.273014098405838, -2.410569906234741, -0.6460942029953003, -0.035966988652944565, 0.24349361658096313, 2.0387046337127686, -1.021526575088501, 0.17816156148910522, -1.2423038482666016, 1.0487674474716187, -0.11315355449914932, -0.2950010895729065, -2.2678487300872803], [-1.081526517868042, 0.04791302606463432, 0.04132792726159096, 1.0531655550003052, 1.5400553941726685, -1.5275547504425049, 1.1251686811447144, -1.098709225654602, 0.5612401962280273, 1.2927011251449585, 0.5246246457099915, -0.9797285795211792, -1.138528823852539, 1.4125488996505737, -0.0675521194934845, -3.084489345550537, 0.9863253831863403, -0.84250807762146, -1.2822531461715698, -1.8793431520462036, 0.2596978545188904, 0.8780404329299927, 0.13647598028182983, -0.13711757957935333, -1.3238276243209839, -0.6392645239830017, -0.22721943259239197, -1.0486661195755005, -0.9310252070426941, 0.500286340713501, -1.220815658569336, 1.163573145866394, -0.370231956243515, -0.9036391973495483, -0.6362432241439819, 0.9949620366096497, -0.371695339679718, 0.0092695914208889, 0.6897396445274353, 1.022669792175293, 1.1838109493255615, -1.1390379667282104, 1.544914722442627, -0.04534444957971573, 0.265318900346756, 0.055691689252853394, -0.6362968683242798, -0.3561178743839264, -1.967560052871704, -0.5005378723144531, -0.28939470648765564, -0.20420575141906738, 0.16615809500217438, -0.29985883831977844, 1.6903141736984253, -1.2492859363555908, 0.2859823405742645, 0.7380738854408264, -0.17770372331142426, -1.5337803363800049, -0.01871579699218273, -0.0058323536068201065, -2.1363203525543213, -1.776632308959961, -0.9757106900215149, 1.7297621965408325, -1.167330026626587, -0.2153216451406479, -1.545021891593933, -1.0365610122680664, 0.11200041323900223, -0.6709129214286804, 0.44363951683044434, 1.9461995363235474, -0.17095516622066498, 0.6045514941215515, -0.14421339333057404, -2.045668840408325, 2.4575424194335938, -0.30841580033302307, 1.3921598196029663, 0.2798124849796295, -0.7740837335586548, -2.2329647541046143, 1.6656123399734497, -0.9719530940055847, 0.5773721933364868, -0.7997457981109619, -1.2756857872009277, -1.4464370012283325, 0.006738088559359312, -0.9101793766021729, -1.6954280138015747, -2.510356903076172, 2.561802864074707, 1.1858879327774048, -1.4945390224456787, -0.24664510786533356, 0.07486917823553085, -1.5578638315200806, 0.949835479259491, -0.4517204463481903, -0.80040442943573, -0.5705481767654419, -0.6249873638153076, 0.09633627533912659, 0.5628093481063843, 1.1218866109848022, -0.29769691824913025, 1.5302050113677979, 0.27544495463371277, -1.3283014297485352, 1.5714002847671509, 0.18159590661525726, 0.31770098209381104, 0.279092013835907, 0.1708674430847168, -0.8438053727149963, -0.7154853940010071, 1.483041763305664, -0.5874866247177124, -0.4113530218601227, -0.9499079585075378, -1.2444145679473877, 0.36596372723579407, -0.10738059878349304, 0.5976548194885254, -1.213290810585022], [-0.97891765832901, 0.012542853131890297, -0.5309109687805176, -1.312806487083435, 1.181004285812378, 1.038059115409851, -1.0314027070999146, 1.6038967370986938, -0.6295678019523621, -0.688546895980835, -0.5123136639595032, -0.5854802131652832, -0.5845038890838623, -0.20586422085762024, 1.586549162864685, -0.6377279162406921, -0.012649618089199066, 0.8488152027130127, 0.31489598751068115, 1.420645833015442, -1.0854357481002808, -0.25093311071395874, 0.8224037885665894, -0.14856526255607605, 1.1943279504776, 0.2016373723745346, -1.9206475019454956, -1.4513932466506958, -0.3826809823513031, -0.6804554462432861, 0.7883105278015137, 0.3950914740562439, 0.5452485084533691, 0.03931621089577675, 0.3600934147834778, 0.0704132542014122, -0.19647745788097382, -0.18177242577075958, -0.3398256003856659, -0.3041257858276367, 1.3649888038635254, 0.7912195324897766, -0.472665399312973, 2.0685880184173584, -0.4126012623310089, -0.39943477511405945, 0.4762687385082245, 0.43519556522369385, -0.8180692791938782, 1.5061715841293335, 0.39779379963874817, -0.2314598709344864, -1.0937318801879883, -0.09490357339382172, -0.1155581921339035, -1.785019874572754, -0.979824423789978, 0.6068147420883179, -0.48032405972480774, -0.02812812849879265, -0.4418031573295593, 1.6213918924331665, -0.9249113202095032, 0.8253446221351624, -0.5817055106163025, 0.2127489298582077, -0.2881511151790619, -0.36073416471481323, -0.29495975375175476, -0.4536573588848114, 0.0936087891459465, -1.0232995748519897, -0.8846229910850525, 0.7791434526443481, -0.7459148168563843, 1.1554288864135742, -0.5326762795448303, -0.4286350905895233, -0.3758213520050049, -1.2441222667694092, 0.549078106880188, -0.7463066577911377, -1.1491074562072754, 0.12304078787565231, -0.35012510418891907, 0.4239519536495209, -0.6289421319961548, 0.18882329761981964, 0.16521242260932922, 0.27115756273269653, -1.0587385892868042, -2.40598464012146, 0.2736189365386963, -0.8392221331596375, -1.3671995401382446, -0.8501628041267395, 0.4142998456954956, -0.06634525954723358, 0.8898608684539795, 0.5249823331832886, -0.6758100390434265, -0.7849746942520142, 0.1057620570063591, 0.03994918242096901, -0.14292116463184357, 0.026161380112171173, -0.13443706929683685, 0.6594139933586121, 0.20500041544437408, -1.528501033782959, -2.5218281745910645, 0.6573927402496338, -0.4621742367744446, -0.2418748289346695, 0.31396275758743286, 0.517876386642456, 0.36134278774261475, 1.6740798950195312, 0.6416143178939819, -0.2230956107378006, 0.6872773766517639, 1.4516664743423462, -0.4044645428657532, -1.3360203504562378, -0.3247254192829132, -0.158417746424675, -0.8232954740524292, 0.6053386330604553], [1.1522083282470703, 0.23766590654850006, -1.007510781288147, 1.6117637157440186, -1.9281424283981323, -0.44452109932899475, 0.7122809886932373, 0.02123354934155941, -0.6020156145095825, -1.5602796077728271, -0.3099902868270874, 1.1952152252197266, -1.1065393686294556, -1.0738416910171509, 0.9644198417663574, 0.934431791305542, 0.5488131642341614, 0.09323418140411377, 0.2524871528148651, 0.4428134858608246, -0.34981241822242737, -0.027468806132674217, -2.770519971847534, 1.0932011604309082, -1.4426994323730469, -1.0798097848892212, 0.039896875619888306, 0.5597468614578247, -0.27510136365890503, 0.214810311794281, 0.976249098777771, 1.4727593660354614, -0.008117209188640118, 0.6823107004165649, 0.8887091875076294, 0.9852649569511414, 2.407097101211548, 0.7901231646537781, -1.1300121545791626, -0.5573325157165527, -1.347179889678955, 0.029068728908896446, -1.5404218435287476, -0.2550828158855438, 1.938854694366455, -1.2411667108535767, -0.6510955691337585, -0.1562599092721939, -0.07743606716394424, -0.2456401288509369, 0.5364480018615723, -0.7135031223297119, -0.43308210372924805, -0.07953181117773056, -0.2181631326675415, -0.6762747764587402, 0.10307929664850235, -0.05553397536277771, -0.4080047011375427, 0.282360315322876, 0.9993587136268616, 0.10788509249687195, -0.3341526985168457, 0.4471781551837921, -0.5702682137489319, -0.3291975259780884, 0.4692729711532593, -0.859216570854187, 1.6668671369552612, 0.5778694152832031, -1.8595521450042725, 1.5957063436508179, 1.1907763481140137, -1.767338514328003, 0.2154482752084732, 0.2144063413143158, -0.697618305683136, -0.22242745757102966, 0.9935781955718994, 0.4075568914413452, -0.43243467807769775, -1.5065381526947021, -0.14054492115974426, 1.0015404224395752, -2.6130282878875732, -0.010108706541359425, 0.3498665690422058, 0.6385656595230103, -1.4465802907943726, -0.06216927617788315, -0.6413744688034058, 0.7105388045310974, 2.465214967727661, -0.4248477816581726, 1.0941121578216553, 1.3878620862960815, 1.5413495302200317, -0.7445233464241028, -0.13125766813755035, 0.5213903784751892, 1.662820816040039, 0.4248719811439514, 0.42675328254699707, -0.8694992065429688, -0.8476598262786865, -1.1213641166687012, 1.13969087600708, 1.4093397855758667, -0.7533064484596252, 0.13036523759365082, 0.3194355368614197, -1.056871771812439, -1.1657726764678955, 1.2814404964447021, 0.2103615403175354, -1.0561400651931763, 0.41728487610816956, -2.3116002082824707, 0.48381465673446655, 0.38531896471977234, 0.20112505555152893, -2.279003381729126, 2.2411587238311768, 0.7154986262321472, 0.15288488566875458, -2.517172336578369, 0.14842373132705688, -0.41290083527565], [-1.4305213689804077, 0.9850857853889465, -1.544784665107727, 0.25221455097198486, 0.6928251385688782, -0.28183767199516296, 0.24134331941604614, -1.6159807443618774, 0.6326747536659241, 0.40870580077171326, 0.8728035688400269, -1.5548584461212158, -1.0759022235870361, 0.254456490278244, -1.0172393321990967, -1.2138382196426392, 0.22162757813930511, 0.10895504802465439, 1.7055046558380127, 1.7820059061050415, -1.105664610862732, -0.12568068504333496, -0.10349558293819427, -0.3566524088382721, -1.0860803127288818, -0.5889172554016113, -1.3340851068496704, -0.9627133011817932, -0.21417731046676636, -0.8274890780448914, 1.374119520187378, 0.14314736425876617, 0.997556209564209, -0.5106388926506042, -0.26499462127685547, 0.2563377916812897, 0.6260653138160706, 0.7880568504333496, -0.4621488153934479, 0.8252560496330261, -0.9928781986236572, -0.6266016364097595, -0.35859835147857666, -0.23770193755626678, -0.8373081684112549, 0.8321434855461121, 0.3651781678199768, 1.1259416341781616, 2.1785054206848145, 0.27305567264556885, 1.3297805786132812, 1.3513885736465454, 0.1740744709968567, -0.7438289523124695, -0.022138996049761772, 0.06416787952184677, 0.7726846933364868, 0.9588916897773743, -0.17757979035377502, -2.222264051437378, -1.2628347873687744, -0.6779358983039856, -0.09648048877716064, -0.6852125525474548, 1.3495054244995117, 0.14131926000118256, 0.5629435777664185, 1.617803692817688, 0.9431907534599304, 0.03041648305952549, -0.5778833031654358, 0.4761662185192108, 0.6376456618309021, -2.4138450622558594, -1.0059996843338013, -0.8835749626159668, -2.923862934112549, -1.7514111995697021, -0.40796342492103577, 0.18855559825897217, -1.5464200973510742, 1.332858681678772, -0.20168545842170715, -1.9531792402267456, -0.043250955641269684, 0.018481530249118805, 0.24522174894809723, 0.19219833612442017, -0.8234087228775024, 1.3746039867401123, 0.018471164628863335, 0.3261876404285431, -0.009924253448843956, -1.024097204208374, -0.8463777899742126, 0.27950963377952576, -1.0163142681121826, 0.15146292746067047, 0.8682944178581238, -0.7000361084938049, -0.6470626592636108, -0.645033061504364, -1.786228895187378, 0.3202599287033081, 1.0517781972885132, -0.34195294976234436, -1.9185889959335327, -0.5417255163192749, -0.37738457322120667, -0.49926134943962097, 0.801876962184906, 0.600981593132019, 2.7864644527435303, 0.4787645637989044, 1.0833088159561157, -0.13861916959285736, -1.2686071395874023, 0.39350661635398865, -0.41200023889541626, -1.3227792978286743, -0.6082165837287903, -1.1422399282455444, 1.2656306028366089, -0.46022963523864746, 1.5655258893966675, -1.8981692790985107, -1.8771010637283325, 1.540131688117981], [1.0157268047332764, 1.6722428798675537, 0.6567541360855103, -0.321544885635376, -0.4934207499027252, -1.9168717861175537, 0.022244486957788467, 0.3726098835468292, -0.7806148529052734, -0.05195047706365585, 0.11240163445472717, 0.4768267273902893, 0.2400268018245697, -2.9405388832092285, -0.10653506964445114, 0.15988749265670776, 0.2574307918548584, 1.3251006603240967, -0.9336191415786743, -0.019938016310334206, 0.7718114256858826, 0.7487861514091492, -0.5887768268585205, 1.5272390842437744, -0.3252948224544525, 0.9448169469833374, 1.0333815813064575, 0.036580368876457214, 0.7960829138755798, -1.1564719676971436, -1.5148464441299438, 0.012946579605340958, -0.7332822680473328, 0.7613650560379028, -0.379823237657547, 0.44759267568588257, -0.22202308475971222, -1.7727409601211548, 0.005182971712201834, 0.7041777968406677, -0.016591321676969528, -0.26878100633621216, 1.1752986907958984, -0.25763365626335144, -0.9973577857017517, -0.3580497205257416, 0.7865586280822754, 0.9244436621665955, 0.7368675470352173, -0.8003333806991577, -0.8787681460380554, -0.9947075247764587, -0.42657670378685, -1.146655559539795, -0.6910462379455566, -0.4817739427089691, 0.3291272819042206, -1.6741865873336792, 1.0675084590911865, 0.04439152404665947, -0.8985655903816223, 1.160575270652771, -1.6142189502716064, -0.14518854022026062, 0.3354414999485016, 0.3754796087741852, -1.4635519981384277, -0.9237402677536011, 0.7889647483825684, 0.6054482460021973, 1.0945266485214233, 2.158247470855713, 0.23842157423496246, -0.01369011215865612, -0.694153368473053, -1.3811291456222534, 1.29851496219635, -0.6873941421508789, 0.6837316751480103, -1.2953894138336182, 0.9174986481666565, -0.5716384053230286, -1.1036908626556396, -1.7021827697753906, 0.9205582141876221, -1.1350346803665161, 0.33299192786216736, 0.25231853127479553, 0.6922970414161682, -0.10993429273366928, 0.41527238488197327, -1.0259631872177124, 0.4121951162815094, -0.23510341346263885, 1.3343064785003662, -0.79063880443573, -0.3479584753513336, 0.5755070447921753, 0.02059665136039257, 0.8182732462882996, -0.3407808542251587, -1.34990394115448, 0.20357485115528107, -1.5067802667617798, 0.8987526893615723, -0.5451750755310059, 0.3171685039997101, 0.47307875752449036, -2.627528190612793, 0.3991156816482544, -1.2415214776992798, -2.336303234100342, 0.20596036314964294, -0.6137285828590393, 0.6545587182044983, -0.33153900504112244, 1.2733007669448853, -0.7721153497695923, 0.3714869022369385, -1.5300897359848022, 0.9766786098480225, 1.0001132488250732, 2.720417022705078, -0.6896906495094299, 0.17011898756027222, -1.6596146821975708, -0.4936509132385254, 0.7731620073318481], [0.8572882413864136, 0.7996627688407898, 1.237614393234253, -0.6883454918861389, 0.3907584249973297, 1.3524948358535767, -1.0795027017593384, 0.8905531764030457, -0.34360024333000183, 0.2629343271255493, 0.06459987163543701, -0.20058603584766388, 0.9629635214805603, -0.8728265166282654, 0.7348907589912415, 1.4191687107086182, -0.34103676676750183, -0.8849092125892639, -0.5092291235923767, -0.011065891943871975, 0.06619185954332352, 0.8319929242134094, -0.30551379919052124, 1.5912691354751587, -0.7005488276481628, 1.3255456686019897, -0.7190083861351013, 0.9649685621261597, 0.28811535239219666, 0.19601592421531677, -0.9972835779190063, -1.1634193658828735, -0.0036925103049725294, 0.5447530150413513, 0.3121466636657715, -0.10754667222499847, 1.379296064376831, 0.48535487055778503, 0.9736064672470093, 0.18523797392845154, -0.12400510907173157, 0.04643448442220688, 0.4674699306488037, 0.40162143111228943, 1.2397220134735107, -1.022302508354187, 0.21277053654193878, -0.20547419786453247, -0.485539048910141, 0.2624278664588928, 0.7278181910514832, 1.2472583055496216, -0.012671149335801601, -0.18344931304454803, 0.997628390789032, 1.2898372411727905, -1.7161961793899536, 1.521254539489746, -0.3612043559551239, 2.0480875968933105, 0.34844839572906494, 0.9063772559165955, -0.26470857858657837, -0.28134989738464355, 2.8547897338867188, -0.3250826895236969, -0.7043558359146118, 0.9662269949913025, -0.03457478806376457, -0.5369156002998352, -1.1487749814987183, -0.9209675788879395, -0.34329453110694885, -0.020615125074982643, 0.22158659994602203, -0.3602673411369324, -0.8447570204734802, 1.042002558708191, 0.24346986413002014, -0.9013445973396301, 0.7308520078659058, 1.3635023832321167, -1.8958176374435425, -0.9432766437530518, -0.2129642814397812, -0.7299814820289612, -0.8898531198501587, -0.12257307767868042, 0.8600631952285767, 1.3468642234802246, 0.5970821380615234, -0.4307743310928345, 0.9293835163116455, -0.6240617036819458, -2.5475401878356934, 0.3291270136833191, -1.1066455841064453, -0.3512503206729889, 1.1628199815750122, 0.01791311241686344, -0.15805354714393616, -0.2929477393627167, 1.7305278778076172, -0.6746214628219604, 0.21171920001506805, 0.5954041481018066, 0.8300994634628296, 0.6106204986572266, -0.8765093684196472, 2.7238242626190186, -0.48485592007637024, 1.744857907295227, 0.6190007925033569, 1.0488907098770142, -1.4451370239257812, 1.1908537149429321, -1.0414416790008545, -1.203573226928711, 0.8059486150741577, 0.09080152958631516, 0.2632005512714386, 1.5781605243682861, 0.8072377443313599, -1.7876356840133667, -1.0253815650939941, -0.39717456698417664, -0.2102382779121399, 1.1179699897766113], [0.32243064045906067, -0.378192663192749, -0.5285894274711609, -2.128201723098755, 0.36218076944351196, -0.1197495311498642, 0.4167740046977997, -0.48233863711357117, 0.6399008631706238, 1.7297767400741577, -0.4308115541934967, 0.49929824471473694, 0.7323814034461975, 0.596572995185852, 0.38938164710998535, -0.7095112800598145, -2.0533103942871094, 2.55928111076355, -1.600387454032898, 0.03995034471154213, -1.3589903116226196, -1.7996573448181152, 1.0530210733413696, -0.06107249855995178, 0.6248251795768738, -0.031178191304206848, -0.5748886466026306, 0.7802256345748901, -0.1338592767715454, -0.37785711884498596, 0.9179219007492065, 0.5981959104537964, -1.0081267356872559, -0.3245401084423065, 1.4998825788497925, -0.25415945053100586, 0.562403678894043, 0.3057151734828949, -1.5419435501098633, -0.7092097401618958, -0.8267688751220703, 0.6362596154212952, -0.5137919187545776, 0.841139018535614, -1.4543633460998535, 0.621470034122467, -0.5309397578239441, 0.004225737880915403, -0.0996021032333374, 0.24589803814888, -0.40481600165367126, -0.0942574143409729, 0.5062688589096069, 0.9795712828636169, -0.21020659804344177, -1.0328763723373413, -0.3572502136230469, 0.9195497035980225, 0.33659082651138306, -1.400484323501587, 1.514973759651184, -0.5288515686988831, -0.8521987199783325, -0.10685118287801743, -1.2089110612869263, -0.805338978767395, -0.8262713551521301, -0.7164143919944763, 0.5654771327972412, -1.6238433122634888, -0.26373475790023804, -0.5520431399345398, -0.30549904704093933, 1.804882287979126, 0.12767574191093445, -0.9110307097434998, -0.159064382314682, 0.5856339335441589, 0.31236031651496887, 0.7808414101600647, 1.4542726278305054, -1.1497999429702759, 1.0675803422927856, 0.3231244385242462, 0.25791943073272705, -0.8689178824424744, -2.3134348392486572, 1.492105484008789, 0.030557911843061447, 2.1233129501342773, -1.1163232326507568, 0.006842960137873888, -2.7841897010803223, -0.21164391934871674, 0.6508018374443054, 0.05823246389627457, 0.8148357272148132, 0.39887478947639465, 0.9036808609962463, 0.16721321642398834, 1.810595154762268, 1.3260000944137573, -1.0241373777389526, 1.2774959802627563, -0.9749280214309692, 0.8037851452827454, 0.7373956441879272, -1.0700948238372803, 0.17082151770591736, 1.1818609237670898, 0.4293144941329956, 1.7261645793914795, -0.21056398749351501, 0.9324939846992493, 0.29033198952674866, -0.10694015771150589, -0.1057049110531807, -0.7792219519615173, -1.6095755100250244, 0.8555065989494324, 0.08767063170671463, 0.5385749340057373, 1.3211240768432617, -0.8777473568916321, 0.26867035031318665, 0.8849883079528809, 0.10455562174320221, 0.24707430601119995]]], [[[0.5995078682899475, 0.1648818850517273, 0.3218200206756592, 0.1612737476825714, -0.7421958446502686, -0.7038698792457581, 0.4450496435165405, 0.9813508987426758, -2.03399920463562, -1.3960578441619873, 0.9516288042068481, -0.32696110010147095, 0.6220279932022095, -1.154524564743042, -0.26291555166244507, 0.5378345251083374, 0.42926570773124695, 0.6080073118209839, 0.23938409984111786, 0.8174072504043579, -1.2939748764038086, -0.5401527285575867, 1.022939920425415, 1.374459147453308, 0.2568156123161316, -1.1587249040603638, -0.2149958312511444, -0.1815100461244583, 0.3774375021457672, -0.6607845425605774, 1.1646252870559692, -0.18610107898712158, -0.3530912697315216, 0.42036983370780945, 1.2595407962799072, -0.13037748634815216, 0.3761506676673889, -0.6404927372932434, -0.005039147101342678, -1.5638498067855835, -1.3256696462631226, 0.4522448480129242, -0.05860090255737305, 0.09067658334970474, -0.9590337872505188, -2.0655291080474854, 0.7302647829055786, 0.37185540795326233, -0.45221447944641113, 1.182083010673523, -0.534521758556366, -1.1722050905227661, -0.7596557140350342, 0.39352333545684814, -0.2754502594470978, -0.19680854678153992, -0.173846036195755, 0.9580584168434143, -1.0941681861877441, 0.4156127870082855, 0.9996488094329834, 0.18946708738803864, -0.4950171113014221, -0.2966822683811188, -0.26916128396987915, 0.9415962100028992, -0.17319001257419586, -0.2654063403606415, 0.39226609468460083, 0.557803750038147, 1.2139092683792114, 1.079525351524353, -0.007599096745252609, 0.7299219369888306, -0.9297635555267334, -0.34390994906425476, -1.1138365268707275, -0.8809055089950562, 0.5337730646133423, -1.2255789041519165, 0.23405855894088745, 0.03744484484195709, 0.1421358436346054, -1.7883665561676025, -0.884986400604248, -0.7986636161804199, -0.28222206234931946, -0.012858418747782707, -1.2638850212097168, -0.25250986218452454, -0.5081337094306946, 1.456284999847412, 0.15609802305698395, -0.45739591121673584, 0.7370800971984863, 0.5353849530220032, -1.0862752199172974, 0.26009702682495117, -1.2709418535232544, 0.03372307866811752, 0.011808730661869049, -0.5403881072998047, -1.275639533996582, 1.4582054615020752, -0.2683977782726288, 0.5848504304885864, 0.25334861874580383, 1.6423251628875732, -2.3532230854034424, -0.12563759088516235, -1.0289075374603271, -0.9212252497673035, -1.6619513034820557, 1.8903759717941284, 0.1336168795824051, 0.9546483159065247, -0.3428581655025482, -1.716736078262329, 0.872793436050415, 0.3617023527622223, -0.4340190291404724, -0.5189029574394226, -0.682681143283844, 0.048288844525814056, -0.20490801334381104, -0.4513283967971802, -0.4453660249710083, -1.2316250801086426], [2.3461997509002686, 1.0192525386810303, -0.5976314544677734, 1.37975013256073, 1.3037017583847046, -1.1364669799804688, -0.3226187527179718, -0.4092380404472351, 0.1921456754207611, 1.3914039134979248, -0.5695158839225769, -0.06202302500605583, 0.3962736129760742, -0.1270328313112259, 1.169136881828308, 0.41064271330833435, -1.3992749452590942, -2.8574225902557373, -0.8850907683372498, -0.4242760241031647, 0.2001774162054062, 1.2670025825500488, -0.46811941266059875, -1.4202821254730225, 0.9676320552825928, 1.0934031009674072, 0.09144093096256256, -1.8039549589157104, 0.6969355344772339, 0.6867433190345764, 0.2792593240737915, 0.662922739982605, 0.18397679924964905, 0.33681946992874146, 0.21716725826263428, 0.4678262770175934, -0.22594797611236572, 0.2476108968257904, 0.762770414352417, 0.029761048033833504, 0.6077517867088318, -1.9793121814727783, -0.6076171398162842, 0.7437456250190735, -0.39097854495048523, -1.087668776512146, 0.4090336859226227, -1.2294273376464844, 0.1285635232925415, 0.10068203508853912, 1.2067350149154663, 0.58329176902771, 1.5315738916397095, 0.3273860812187195, -0.7669020891189575, 0.8485268354415894, -1.449347972869873, -2.0425758361816406, -0.3241274952888489, 0.18940535187721252, -0.24173910915851593, -0.5807923078536987, -0.07158984988927841, 0.2667216360569, -0.9840983152389526, -1.3448532819747925, 0.9293647408485413, 0.29094398021698, -1.1829087734222412, -0.9666266441345215, 1.4674201011657715, 1.6500871181488037, -1.2447742223739624, -0.47544410824775696, -0.9472091197967529, -0.4510653018951416, 1.3696521520614624, -0.8141410946846008, -0.36051082611083984, 0.09166762232780457, 0.038330115377902985, 0.35393989086151123, -1.8661746978759766, 1.9936131238937378, -0.0350794643163681, -0.013290992006659508, 0.878431499004364, -0.3033393323421478, 0.450369656085968, 1.1914856433868408, -0.6579548120498657, -1.1788444519042969, 0.3859599828720093, 0.822567343711853, -0.06420005857944489, -1.088841199874878, -1.4517518281936646, -0.5610272288322449, 0.8850759267807007, 0.39889028668403625, -0.17200177907943726, 2.2181286811828613, -0.12692482769489288, 0.712270975112915, 1.914811611175537, 1.00777006149292, -0.6578827500343323, -0.417740136384964, 0.7496523261070251, -0.3145521283149719, 0.4499501585960388, 0.9701765775680542, -1.0523427724838257, -0.8021894097328186, -1.2279069423675537, -0.1746026873588562, 2.1556999683380127, 0.28221559524536133, -1.390270471572876, 2.5841026306152344, 1.3580529689788818, 1.5937830209732056, -0.07280457019805908, -0.5292835235595703, -0.18130676448345184, 0.46302908658981323, -0.23225797712802887, -0.2634059488773346], [1.5509133338928223, -0.4088233411312103, -1.1792017221450806, -0.7303470373153687, -0.9568713307380676, -1.3221594095230103, 1.372605562210083, -1.435500144958496, -0.5430478453636169, -1.1104772090911865, 0.9267385005950928, -1.1927385330200195, 0.06007115915417671, 0.5172820687294006, 2.182302236557007, 0.9638048410415649, 0.7100624442100525, 0.6176326274871826, 0.7774367928504944, 0.4472617208957672, 0.3907918334007263, 0.39254069328308105, -1.20896315574646, 0.9459917545318604, -0.9911895394325256, 0.6094961762428284, 0.4057534635066986, 0.6816701889038086, -1.8703858852386475, 0.08586645126342773, -0.2397281676530838, 0.6422147750854492, 0.7106751799583435, 0.6736049652099609, -0.2742319703102112, -0.24517302215099335, 0.5003466606140137, -2.232921838760376, -1.5746822357177734, 0.2504943013191223, 0.29207637906074524, -0.7165873646736145, -0.6380119323730469, 0.871015727519989, -0.16060329973697662, 0.687591016292572, -0.9414569735527039, 0.9349985718727112, -0.12298671156167984, 0.2666158974170685, -0.22170650959014893, 0.015167471952736378, 1.4958477020263672, -0.9509198069572449, -0.10673701763153076, -0.8119942545890808, -0.003184005618095398, -0.5332456827163696, 1.5592067241668701, 0.05152932554483414, -2.682288885116577, -1.2228730916976929, -0.19254006445407867, 0.018834313377738, 1.5373125076293945, 0.1451224535703659, -0.011825411580502987, -0.821588933467865, 0.6131930947303772, -0.6259480714797974, -0.5250438451766968, -0.37446820735931396, -1.0511753559112549, 0.3599584996700287, -0.12286719679832458, 0.02373071201145649, -1.472782850265503, -0.4482850432395935, -0.016322501003742218, -1.843096137046814, -0.14446911215782166, 0.16997657716274261, 0.6293673515319824, -0.21015284955501556, -0.27258676290512085, 0.7389441132545471, 1.8262720108032227, 0.945523738861084, -0.007175952661782503, 1.6789040565490723, -0.6259644627571106, -0.3541829288005829, -0.32252687215805054, -0.6984513998031616, -2.39322829246521, 2.5515477657318115, 0.039179928600788116, 1.0986130237579346, -1.8577075004577637, -0.8926022052764893, 0.23250792920589447, -1.4419535398483276, 1.2120747566223145, 0.15498460829257965, -1.4861027002334595, -0.1434655636548996, -1.0109366178512573, -0.002647290239110589, 0.6658518314361572, 0.8996400833129883, 0.7846612334251404, -0.9066821932792664, 1.290073037147522, 0.11678565293550491, -0.7188636064529419, 0.7685095071792603, 0.21938002109527588, 1.176146388053894, 0.12321033328771591, 0.8759140968322754, 0.905045747756958, -0.5941675901412964, -0.4459609091281891, 0.7356007695198059, 1.7146579027175903, 0.36556223034858704, 2.061692953109741, -1.3934861421585083], [-0.5992990136146545, 1.2748048305511475, 0.5145545601844788, 1.564502239227295, -0.36686795949935913, 0.5043559670448303, -0.7154390811920166, -0.25293272733688354, 0.742655873298645, 0.7083519101142883, 0.4458465576171875, -0.8566543459892273, 0.6765882968902588, 0.3796023726463318, 1.4641246795654297, -1.7315552234649658, 0.1652609407901764, -0.938738226890564, -0.4952794313430786, 2.622936964035034, -0.8064431548118591, -0.9155495762825012, -0.49441981315612793, -1.2615621089935303, 0.5583376884460449, -0.20230333507061005, -0.7620480060577393, 1.5369287729263306, -0.554223895072937, -1.3001508712768555, -0.6135838031768799, -1.636902928352356, -1.07183039188385, -1.6643270254135132, 0.2183224856853485, -0.11821155250072479, 0.1188802719116211, 1.0451819896697998, 0.6926683783531189, -0.3240891396999359, 0.8686485290527344, 0.9950414896011353, 1.681060552597046, -2.04606294631958, -0.488278865814209, 0.06564248353242874, -1.5231949090957642, 0.8948668837547302, -0.6415082812309265, -2.3313112258911133, 0.7419096827507019, -1.4078775644302368, 1.1310642957687378, 2.1381466388702393, -1.6518124341964722, -0.6321909427642822, 0.19227316975593567, 1.7175853252410889, -0.21456179022789001, -0.6863914728164673, 0.9960347414016724, -1.3235325813293457, 0.17402517795562744, 1.535400629043579, 0.9811500310897827, -0.08160707354545593, -0.04023955389857292, 0.8292496204376221, 0.8114244937896729, 1.549248218536377, 0.29394522309303284, -0.1899895966053009, -0.8432779908180237, 0.02785089798271656, 1.9345099925994873, 0.7597634196281433, 0.8252726197242737, -0.09550265967845917, 2.763277769088745, -0.8247602581977844, -0.666990339756012, 2.413231372833252, -0.307574987411499, 1.1816074848175049, -0.31314653158187866, -0.9533395767211914, -1.4800453186035156, -0.3073732554912567, 0.1931874006986618, 0.07532138377428055, -0.08569446951150894, 0.61114501953125, -1.4676473140716553, 0.8799803256988525, 1.2008429765701294, -0.2891683280467987, -1.4481221437454224, -1.451750636100769, 0.11266475915908813, 1.490373134613037, -0.22035138309001923, -0.7258040308952332, 1.3792415857315063, 0.2622178792953491, -0.8933107256889343, -1.6917564868927002, 1.8607367277145386, -0.1429203301668167, 0.32667678594589233, -0.8055523037910461, 0.13969869911670685, -0.2042837291955948, 1.5662328004837036, 1.3562909364700317, -0.49948325753211975, -1.5257645845413208, 0.17131690680980682, -0.26814013719558716, -0.8108484148979187, 1.4547641277313232, 0.23143678903579712, 0.28711220622062683, -1.9799779653549194, 0.9268427491188049, -1.5959298610687256, 0.2740066349506378, 0.18682238459587097, 0.01614036038517952], [-0.36928561329841614, 1.0515828132629395, -1.0523247718811035, 1.4353855848312378, -0.801820695400238, -1.3533737659454346, -1.023819923400879, 1.1719143390655518, -0.11478880792856216, 1.7130898237228394, -0.05405798554420471, 0.8815044164657593, 0.4255404472351074, 0.3875759243965149, -0.14497175812721252, -0.9027343392372131, -1.011674165725708, -0.8718203902244568, 1.433534026145935, -0.41860494017601013, 0.5473690032958984, 0.12221582233905792, -0.18775585293769836, 1.130670428276062, 0.9357638955116272, -0.7111692428588867, -1.086728572845459, -1.0419185161590576, 1.1428453922271729, 0.3930809795856476, 1.154422402381897, 0.23233434557914734, 1.0343835353851318, 0.2452840358018875, 0.10246249288320541, 1.273542046546936, -0.38697630167007446, -0.7060171961784363, 0.9422938227653503, -1.2737653255462646, -0.6203479766845703, -1.1142914295196533, 0.13921289145946503, 0.648474395275116, -1.2379597425460815, 0.21205301582813263, 0.40147310495376587, -0.38523581624031067, 1.1732245683670044, 1.1598891019821167, 3.0642988681793213, 0.30499327182769775, 0.5831247568130493, -0.5189635157585144, -0.2559695541858673, 0.9873115420341492, -0.9933817386627197, 1.0811614990234375, -0.3862614035606384, 0.24680940806865692, 1.693945288658142, -1.2348593473434448, 0.5792966485023499, -0.26024335622787476, 1.021239995956421, -0.7905292510986328, 1.3170456886291504, -0.2550060749053955, -0.2387048453092575, -0.8963600993156433, -1.2589223384857178, 0.40592119097709656, 0.9030691385269165, -0.01553992461413145, 0.45092323422431946, -0.9356256127357483, -0.31862297654151917, -0.17818692326545715, 0.6326696276664734, 1.0290154218673706, 0.16621339321136475, 1.0578324794769287, -0.46864259243011475, 1.1650817394256592, -0.8052837252616882, 0.18980753421783447, 0.3564006984233856, 1.8184503316879272, 0.9354649782180786, -0.7125222086906433, -0.39191755652427673, -0.811100423336029, -0.7532250285148621, -1.7557270526885986, 1.6261003017425537, -1.7359064817428589, -1.0185240507125854, 0.017781037837266922, -0.15457725524902344, 0.7140457034111023, 0.08175109326839447, 2.1220078468322754, 0.35206517577171326, -1.0087007284164429, 0.35752278566360474, 0.7195801138877869, 0.8885539770126343, -0.3103579878807068, -0.2917259633541107, 0.5588807463645935, 0.02878212183713913, -0.09235715121030807, -0.03409045189619064, -2.61413311958313, -0.30210980772972107, 0.15557771921157837, 0.884389340877533, 0.3647312819957733, 0.5621278285980225, -0.2222280651330948, -1.3417690992355347, 0.7816259264945984, -0.55699622631073, 0.06784561276435852, -0.7710034251213074, -0.7064546346664429, -0.05680381879210472, -0.05225365608930588], [0.027288563549518585, -1.3517969846725464, -0.43096885085105896, -0.7046736478805542, -1.7110904455184937, 0.027366647496819496, -0.8341918587684631, -0.7054854035377502, 0.8416050672531128, 0.6439547538757324, 1.2042325735092163, 0.5642094612121582, 1.195348858833313, 0.76617830991745, 0.5864208340644836, 1.007215976715088, 0.7400023937225342, -1.4216039180755615, -0.12177010625600815, 0.2625866234302521, 1.329067349433899, 1.1855977773666382, 0.11180724203586578, -0.8150615692138672, 1.082115888595581, 0.8263551592826843, -0.013431012630462646, -1.025484561920166, 1.8286864757537842, 0.841049313545227, -0.36413198709487915, 0.704055666923523, -3.260796546936035, -1.4218195676803589, 0.9096952080726624, 0.08811472356319427, 1.4164866209030151, -0.10369681566953659, -1.7100163698196411, 0.4348999261856079, 0.4464566111564636, -1.61928129196167, 0.34971901774406433, 0.6624839901924133, 0.39439889788627625, -1.707951545715332, 0.5755072236061096, 0.8367681503295898, -0.33982276916503906, 0.9921531081199646, -1.5335646867752075, -0.5414762496948242, -0.9986068606376648, 0.27809444069862366, 0.4412578344345093, 0.2508820593357086, -0.987720787525177, -0.9017101526260376, 1.3321924209594727, 0.9750851988792419, 0.08839385211467743, 0.277527391910553, -1.7661120891571045, 1.8198844194412231, -0.355784147977829, -2.0523054599761963, 1.147659182548523, 1.508590817451477, 0.7740835547447205, 1.7376965284347534, 0.40112805366516113, 0.3776989281177521, -0.5249802470207214, 0.7371640205383301, 0.004590928554534912, -0.21204403042793274, -1.8122186660766602, 1.7358688116073608, -1.0430972576141357, 0.8434950709342957, -0.08730199187994003, -0.5019582509994507, -0.9077682495117188, -0.3766458332538605, 0.665820837020874, 0.8475690484046936, 0.25467783212661743, -1.1668040752410889, -1.0779623985290527, 0.32314378023147583, -0.5839933156967163, 0.6252511143684387, -0.5681607127189636, 0.08680345863103867, 1.1071009635925293, 1.3662501573562622, -0.2599261701107025, 0.49959906935691833, -0.0331600122153759, -0.22970843315124512, -0.6284240484237671, 1.5853945016860962, -1.6966036558151245, -0.4337395429611206, -0.26081618666648865, 0.5891833305358887, -0.04596941918134689, -1.1122372150421143, 0.9211476445198059, 0.550150454044342, -2.340301752090454, -0.6812240481376648, 0.6982977390289307, 0.7669174075126648, 0.9695095419883728, -2.0026144981384277, 0.7623383402824402, 1.1041797399520874, 0.8987329006195068, 1.4557918310165405, 0.46247678995132446, -1.9010422229766846, 1.3422861099243164, 0.19175827503204346, 0.21006041765213013, 1.2989752292633057, 1.8088250160217285, 1.1685353517532349], [-0.8115513920783997, 1.1398630142211914, 0.6590143442153931, 0.9627275466918945, 0.7894646525382996, 0.3869327902793884, -0.6127420663833618, -0.25298383831977844, -0.32476234436035156, -1.81737220287323, -0.12027871608734131, 0.15956974029541016, -0.1021212488412857, -0.48880550265312195, -0.21803240478038788, -1.2559959888458252, 0.11468201130628586, 0.03933053836226463, -1.3081903457641602, 2.8978161811828613, 2.2609076499938965, -2.136089324951172, 1.162681221961975, -1.172743797302246, 0.9683697819709778, 0.23358562588691711, -1.6431739330291748, 0.05704067274928093, 0.27211958169937134, -1.6802971363067627, 0.8061966300010681, 0.5414358973503113, -0.6352342963218689, 0.6158905625343323, -0.2960226237773895, 0.24478134512901306, -1.9739000797271729, 0.2453240305185318, -0.34907418489456177, -0.6403005719184875, -0.7581571340560913, -1.7917401790618896, 0.16267737746238708, 0.11905614286661148, 0.7611485719680786, -0.6254178285598755, -0.27366286516189575, -0.060464877635240555, -1.0063313245773315, -1.5145537853240967, -1.205536127090454, 1.9067479372024536, -1.1138062477111816, -0.3161613643169403, -0.12135927379131317, -0.015054022893309593, -1.557104229927063, -0.02631049044430256, -0.5037168860435486, -0.14373213052749634, -0.9681218862533569, -0.6534126996994019, 1.5920847654342651, 0.6994050741195679, -0.9033491611480713, 0.8536738157272339, -0.9729039669036865, 0.79845130443573, 0.771740734577179, -0.8788213133811951, 0.17372392117977142, 0.49038976430892944, -0.9615916609764099, -0.3867886960506439, -0.9145900011062622, -0.2700923979282379, -1.4416471719741821, -0.49846985936164856, 0.8813858032226562, 0.8357378840446472, -0.3768608272075653, 0.5686297416687012, -0.616465151309967, -1.0399575233459473, -0.21818400919437408, -1.002959132194519, -0.8059931993484497, 1.0730572938919067, -0.5992192029953003, 0.3316687345504761, 1.1124420166015625, 0.5046401023864746, -0.8459213376045227, 0.3148123025894165, 0.5421496629714966, -0.3134797513484955, -0.06365025788545609, -0.5631455183029175, 0.5793797373771667, -0.11300349980592728, 0.6757500171661377, -2.565046548843384, -0.2731705904006958, -0.23148711025714874, -0.5219901204109192, 0.719258189201355, -0.6050704717636108, -2.0131847858428955, -1.3971002101898193, 0.4253712594509125, -1.2899819612503052, 1.4696533679962158, 0.8137854337692261, 0.6119306087493896, 0.7472707033157349, 0.7746824622154236, -0.2186814248561859, -0.14569123089313507, -0.5956447124481201, 0.22676947712898254, -1.0838209390640259, 0.5932064056396484, -0.3054535686969757, -1.4998079538345337, -2.1553993225097656, -0.5470283031463623, 0.036719564348459244, -0.788121223449707], [0.762209415435791, -1.0543817281723022, -1.4545903205871582, 1.2694666385650635, -0.6677664518356323, 0.22155971825122833, 1.4472109079360962, 0.7755041718482971, 0.6264969110488892, -0.9939369559288025, -0.18811863660812378, -0.41721096634864807, 1.1858984231948853, -1.327888011932373, 0.8550628423690796, 1.0217667818069458, -1.0910485982894897, -0.17359530925750732, -0.5627091526985168, -0.47670209407806396, 2.022282838821411, 0.47360318899154663, -0.33284881711006165, 0.394025981426239, -1.9750579595565796, 0.8206684589385986, -0.06492075324058533, 0.7445167303085327, -0.005128081422299147, 0.8084065914154053, 1.3443163633346558, -1.074934959411621, -0.8067837357521057, 0.3668036162853241, -0.7262880206108093, 0.2756156921386719, -0.2196359932422638, 0.8686690330505371, 1.1035271883010864, 1.1281589269638062, 0.699733316898346, -0.19993922114372253, -0.3731612265110016, 0.6950775980949402, -1.5861304998397827, 0.6313849091529846, -0.4063189625740051, -1.0957118272781372, 1.6076555252075195, -0.9281134009361267, -3.032243013381958, -0.7145140767097473, 1.1612478494644165, 0.29906511306762695, 0.9648363590240479, 0.1871315836906433, -0.06413204222917557, -1.324698805809021, -0.20552533864974976, -0.22520437836647034, -0.03121507354080677, 0.8813943862915039, 1.3298962116241455, 0.7127177715301514, -0.5627895593643188, 0.8158330917358398, -0.5267888903617859, -0.18337981402873993, -0.20690542459487915, -0.5242032408714294, -0.08018998056650162, -0.39401480555534363, -0.6600357294082642, -1.565652847290039, -0.8196731805801392, -0.18396921455860138, 2.5397982597351074, 1.4424158334732056, 0.5583809614181519, -1.8508899211883545, -0.8313151597976685, 0.44746971130371094, -0.8730681538581848, -0.5518978238105774, -0.40257367491722107, -0.4748680591583252, 1.6969901323318481, -0.529161810874939, -2.1029882431030273, 1.5191820859909058, 0.7839827537536621, 0.6913203597068787, 1.2572153806686401, 0.36048224568367004, 1.783178687095642, 0.5677990913391113, -0.4918239116668701, 0.5628944635391235, 0.24967420101165771, 0.26150816679000854, 0.12916849553585052, -0.04146742820739746, -0.342365562915802, -0.9932342171669006, -1.8825825452804565, -0.6519524455070496, 0.7545960545539856, -0.8671668767929077, -0.012146816588938236, -0.314974844455719, 0.4368763864040375, -0.4963034987449646, 0.9165886640548706, 0.5449766516685486, -0.2947310507297516, -0.026869023218750954, 0.333596408367157, 1.3213708400726318, 0.22892263531684875, -0.16239488124847412, -0.7388628125190735, 0.3889806866645813, 0.3921748995780945, 1.2600980997085571, -1.0613248348236084, 1.3303242921829224, -1.688863754272461, 0.30971741676330566]], [[0.37507104873657227, -0.5575903654098511, 0.2543576657772064, 0.9822249412536621, 0.14251570403575897, 1.541061282157898, 2.1448497772216797, 2.4296839237213135, 0.6779826879501343, 1.321622371673584, 0.30112630128860474, -0.41198965907096863, -1.5232102870941162, 0.7446689009666443, -0.7734702825546265, 0.3920547366142273, 0.043067753314971924, 0.710188627243042, 0.027991198003292084, 1.19808030128479, -0.18221139907836914, -0.9536191821098328, 0.3764960765838623, 1.3600494861602783, 0.5088561177253723, -0.17307494580745697, 1.6030105352401733, 0.6451241970062256, 1.527985692024231, 1.1100053787231445, -0.49342650175094604, -0.21453969180583954, -1.5819675922393799, 1.277125358581543, 2.036181688308716, -0.043417006731033325, -2.0130462646484375, -0.835511326789856, 1.4660638570785522, -0.831935703754425, 0.9508746862411499, -0.1863652914762497, -0.4632752537727356, 0.6176808476448059, 2.0986111164093018, 1.7467641830444336, -1.9374738931655884, 0.44235846400260925, 0.16121385991573334, 0.33479762077331543, -0.6439937353134155, 1.199906587600708, 1.0844051837921143, 0.19899000227451324, 1.2439920902252197, 1.4691038131713867, -0.28011807799339294, 0.1118539571762085, -1.4193153381347656, -1.1138194799423218, 1.6477646827697754, 0.6021167039871216, 0.37966033816337585, -2.466918468475342, -1.3461893796920776, -0.8348926901817322, -1.4116700887680054, 1.110548734664917, -0.6046148538589478, 1.1800744533538818, 0.5274761319160461, -0.832129180431366, 0.07982544600963593, -0.9092229604721069, 0.18889719247817993, 2.208096504211426, -0.6259039640426636, 0.8056881427764893, -2.0809760093688965, 0.1123710498213768, 0.24710002541542053, 0.9283009767532349, -0.027578938752412796, 0.7845785021781921, -0.18340450525283813, -0.6632095575332642, 0.5460186004638672, 0.99764484167099, -1.4922343492507935, -2.1788387298583984, -0.021662039682269096, -2.4887776374816895, 0.6252923011779785, -1.7388235330581665, 0.6545332074165344, 0.5691698789596558, 0.5518009662628174, 0.22817747294902802, -0.39509978890419006, 0.32356691360473633, 0.5968625545501709, 1.1949009895324707, 1.49613356590271, -0.9400961995124817, -0.49417704343795776, 0.18688036501407623, -1.3844952583312988, 0.7040742635726929, -0.38862094283103943, 1.0497866868972778, -0.030957812443375587, 1.536137580871582, 0.17073312401771545, 0.44698721170425415, -0.994929313659668, -0.028488771989941597, -0.25166642665863037, -1.384432315826416, -0.327210932970047, -1.2173594236373901, -0.0041779703460633755, -0.701150119304657, -1.6361302137374878, 0.9653045535087585, -1.513501524925232, 1.0739444494247437, -1.3782029151916504, 1.146426796913147], [0.4276304543018341, 2.2411251068115234, 0.07307619601488113, 0.18927721679210663, -0.5185310244560242, 1.6350244283676147, -0.6162227392196655, -1.2650153636932373, -0.8700589537620544, 2.120802402496338, 1.2249736785888672, 1.1995373964309692, 0.20955874025821686, 0.13605941832065582, -0.7198922038078308, 2.043316602706909, -1.0183466672897339, -0.42668789625167847, -0.3133086860179901, -0.17388425767421722, 0.629813015460968, -1.71226966381073, 1.2851203680038452, -0.8020821213722229, 0.36328160762786865, 1.1894501447677612, 1.2896941900253296, -3.202195644378662, -0.21593959629535675, 0.7194749116897583, -0.4164152145385742, 0.14492729306221008, 0.6365329623222351, -0.741416335105896, -0.48996660113334656, -0.4940235912799835, 0.33602380752563477, -0.4299164116382599, -0.10617250949144363, 0.6669667363166809, -0.41912946105003357, -0.5733587741851807, -1.2644925117492676, -1.161959171295166, -0.9254671335220337, 0.9197455644607544, 1.998971939086914, 0.8579639196395874, 0.10719955712556839, 0.11111552268266678, -1.3763874769210815, -0.008847968652844429, -0.2051994651556015, 0.72513747215271, -0.5707945227622986, -1.1785215139389038, -0.5781501531600952, 0.9810057282447815, -0.806219220161438, 0.15590710937976837, 1.1015795469284058, -0.46939048171043396, 0.7261969447135925, -0.48853495717048645, 0.127830371260643, -0.5215618014335632, -1.000813364982605, -0.33671653270721436, 0.13661253452301025, -0.8971675634384155, -0.22585135698318481, -0.39165860414505005, -0.8816002011299133, 1.093934416770935, -0.9095534682273865, -0.42347678542137146, 0.34876757860183716, -1.1514519453048706, -1.2020103931427002, 1.1072518825531006, -0.040267642587423325, 1.9696919918060303, 0.9594875574111938, -0.1600898951292038, 0.5924155116081238, 0.296699583530426, 2.046013832092285, -0.4568999409675598, 0.3227784037590027, -1.9331920146942139, 0.22619450092315674, -2.332714557647705, -0.16200542449951172, -0.29160434007644653, 0.21024635434150696, -0.3770304024219513, 0.20330923795700073, -2.114656686782837, -1.7504254579544067, -0.06645438820123672, 0.7002445459365845, -0.1561318039894104, -1.0447298288345337, 0.45806044340133667, 0.574874222278595, -0.5151743292808533, -0.2746405303478241, -1.5221209526062012, -0.7004665732383728, 2.0357487201690674, 1.0797407627105713, 1.8532198667526245, -0.3406098783016205, 0.6413727402687073, -0.40957942605018616, -0.9287768006324768, 2.9322991371154785, -0.9276764392852783, 0.7721478939056396, 1.1332907676696777, -0.26632943749427795, -0.42006567120552063, 0.6463987827301025, 0.589766263961792, -1.1022167205810547, -0.7729964852333069, -0.17714980244636536, -0.5274418592453003], [1.5184181928634644, 0.013649996370077133, -1.3810834884643555, 1.0933490991592407, -1.3282430171966553, -2.0786056518554688, -0.7091884016990662, 0.6081928610801697, 1.2007375955581665, -0.09126638621091843, -1.3507144451141357, 0.06128319352865219, 0.22327707707881927, 1.0751293897628784, -1.5794174671173096, -0.3678199052810669, 0.7581079006195068, 0.009969878941774368, -0.30201488733291626, 0.6567955613136292, -0.7657263278961182, 1.399207592010498, -0.2650204002857208, 1.8381913900375366, 1.136672019958496, 0.011590211652219296, -0.392463356256485, 1.535001277923584, -2.5007271766662598, -0.5663425326347351, -0.3433425724506378, 0.921164870262146, 1.3580557107925415, -0.20004235208034515, 0.24908941984176636, -0.5384416580200195, -2.8948519229888916, 0.6997748017311096, 0.2271386831998825, 0.8971970677375793, -0.8357171416282654, 1.0948917865753174, 0.8910509943962097, -0.3370833396911621, -0.1741214245557785, -0.943720281124115, -1.4758719205856323, 0.4622449278831482, 0.30059269070625305, -0.9856783151626587, -2.184380292892456, 1.331681251525879, -0.5941256284713745, -1.008087158203125, 1.3100390434265137, 1.1401253938674927, 0.5267931222915649, -0.41643962264060974, 1.1254503726959229, 0.8491217494010925, 2.1344356536865234, 1.7060242891311646, 1.6114957332611084, 0.5758430361747742, -1.2041493654251099, 1.71964693069458, -0.9511248469352722, 1.2710204124450684, -0.9627959728240967, -3.1009819507598877, -1.5140960216522217, -2.007171154022217, -1.26082444190979, -0.8175815939903259, -2.5163824558258057, -1.6057898998260498, -0.12488431483507156, 1.5621225833892822, -1.7390592098236084, 1.3906162977218628, -0.28024619817733765, -0.7587875723838806, -1.3311889171600342, 0.8015836477279663, -0.22642628848552704, -0.3011776804924011, -0.9697802066802979, -0.04048847034573555, -2.0091381072998047, -0.7109588384628296, 1.2296441793441772, 0.4919637441635132, -0.8499639630317688, -1.1680670976638794, 0.8500310182571411, -0.33746373653411865, -2.1896450519561768, -0.836771547794342, 0.6182937622070312, -0.09112361073493958, -0.665584921836853, 0.9412921071052551, -0.4572061002254486, -1.6735835075378418, 1.0239806175231934, -1.0524401664733887, -0.8399176001548767, 0.6891996264457703, -0.4247450828552246, 1.7993366718292236, -1.401789903640747, 0.606042206287384, 1.739983320236206, 0.08216076344251633, -1.1548995971679688, 0.3073860704898834, -1.7265384197235107, 0.33136439323425293, 1.918932557106018, 0.8495914340019226, 0.6497382521629333, 1.5556656122207642, 0.27669888734817505, 0.14634324610233307, 2.268918991088867, 0.11263308674097061, 1.5900248289108276, -0.7383838891983032], [0.6480459570884705, 0.4207637906074524, -0.8888060450553894, -0.06272489577531815, -1.4823887348175049, 0.45355966687202454, 0.10762803256511688, 0.5687398910522461, 0.942867636680603, -1.9600638151168823, -1.1643280982971191, -0.9723634719848633, 0.11765103787183762, -0.6491073966026306, 0.7643249034881592, 0.03712427616119385, 1.0847687721252441, 0.034003596752882004, 1.01058828830719, 1.6981277465820312, -2.402320146560669, -0.5911250710487366, -0.026828087866306305, -1.5848294496536255, -0.7040013074874878, -0.011687920428812504, 0.210536926984787, 0.6246543526649475, -0.7152550220489502, -0.6951094269752502, 0.9973257780075073, 0.6025581359863281, 0.8558825254440308, -0.3364289104938507, 0.6972851157188416, -0.810609757900238, 0.0494236946105957, 1.46388578414917, -0.28272369503974915, 1.9605659246444702, -0.7543878555297852, 1.4688936471939087, -1.4812530279159546, -0.7647476196289062, 1.1743528842926025, 0.4191935062408447, 1.5957385301589966, 0.5247409343719482, -0.6659165620803833, -0.03132129833102226, -0.06931692361831665, -0.001686077332124114, 0.2879554033279419, -0.0796789601445198, 0.9271527528762817, 0.694789707660675, -0.44709911942481995, -0.809539258480072, 0.2865549623966217, 1.3587044477462769, 1.8136231899261475, -1.4823118448257446, 0.01725110225379467, 0.18181677162647247, 2.4277100563049316, -0.8471546769142151, -0.29962849617004395, 0.4569960832595825, 0.8499102592468262, 0.7701585292816162, -0.212073415517807, -0.5204710364341736, 1.327393889427185, 0.9241991639137268, -0.01635017618536949, 1.1417653560638428, -1.1143547296524048, -0.6270795464515686, -0.3389154076576233, 1.7490183115005493, -0.7463890910148621, 2.422922134399414, 1.3103554248809814, 0.5354172587394714, 0.34052515029907227, -0.21459028124809265, 0.8428077101707458, -2.5005548000335693, 0.08953089267015457, 0.06586681306362152, 0.7116402387619019, 0.7122110724449158, -0.927066445350647, -0.1641012281179428, -0.5672250390052795, -0.4206331968307495, 0.10242359340190887, 0.6964182257652283, -1.232853651046753, 1.5154131650924683, -0.5030217170715332, -0.47589078545570374, -0.18334554135799408, -1.3648103475570679, 0.16091656684875488, -0.00033054931554943323, 0.4611778259277344, 0.06709592789411545, 0.7581786513328552, -0.4201551377773285, 1.2969034910202026, -1.1509679555892944, 0.5477117300033569, 0.8546593189239502, 0.5478782653808594, 0.7256967425346375, -0.679012656211853, -0.7138568758964539, -1.9143000841140747, 0.9012520909309387, 1.3546208143234253, 0.03885743394494057, -0.014536898583173752, 0.6804400086402893, -2.7552459239959717, -0.24757643043994904, 0.49133217334747314, 0.9685760140419006], [-0.15018759667873383, -0.16753050684928894, -0.4604560136795044, -1.183682918548584, -0.9692088961601257, 0.5584844946861267, -0.7986268997192383, -0.6060567498207092, 0.36698150634765625, -1.0350102186203003, -0.7605183124542236, 0.6212794780731201, -0.1007252186536789, -0.500656247138977, -0.7255682349205017, 0.9733703136444092, 0.15387119352817535, -0.5964972376823425, -0.766287088394165, -1.2944473028182983, 0.08565602451562881, -0.197846457362175, -0.0833326056599617, 1.0723600387573242, 0.4644457995891571, -0.5166462659835815, -1.1453887224197388, 1.5388387441635132, -1.8249461650848389, -1.9253698587417603, -0.20442958176136017, 1.057421326637268, 0.9078757166862488, 0.8829938173294067, 0.9718453884124756, -0.650105893611908, -0.18623673915863037, 1.0889812707901, 1.2316100597381592, -1.3468838930130005, 0.014727510511875153, -0.4617830812931061, -0.2985322177410126, 1.3661949634552002, -0.787372887134552, 0.6827859282493591, -1.8983123302459717, 0.5024586319923401, -1.98638117313385, 0.09494256228208542, 0.374401330947876, 1.6673091650009155, -0.3419862687587738, 0.4649469256401062, 0.988544762134552, 0.8246683478355408, -0.20854882895946503, -2.1266379356384277, 0.24600987136363983, 0.435092568397522, -1.039416790008545, 2.4650728702545166, 0.92145174741745, 1.0922967195510864, 1.608741044998169, -0.46719545125961304, 1.3236087560653687, 0.8740896582603455, -1.0112810134887695, 1.0963462591171265, -0.486910343170166, 1.4702060222625732, -0.4960739016532898, 0.2266533225774765, 0.6741242408752441, -0.1254887878894806, -1.0490806102752686, 0.36953145265579224, 0.9255936741828918, -1.5613744258880615, 1.670446753501892, 0.00271525583229959, -1.4446297883987427, -2.1428494453430176, -0.4433177709579468, -0.6433059573173523, -1.2156628370285034, 0.5940017700195312, -0.4801773428916931, -0.3285861611366272, -0.08275307714939117, -0.170780748128891, -0.30225515365600586, 0.1776556819677353, 0.2913203537464142, -1.9706848859786987, -1.2904574871063232, 0.7489926815032959, 0.9673711061477661, -0.5970538258552551, 1.2241092920303345, -1.417694330215454, -0.6813344955444336, 0.2434110939502716, -0.9950539469718933, -0.14907582104206085, 0.5511752963066101, -2.1111185550689697, -0.990155816078186, -1.1833930015563965, 0.6329793930053711, 1.0749328136444092, 0.9341970682144165, 0.9567198753356934, 0.2925699055194855, -0.27938583493232727, -1.4082478284835815, -0.3028646409511566, 0.03455105796456337, 1.0791511535644531, 0.42858636379241943, 1.088240146636963, -0.9533748626708984, 2.0294578075408936, -0.3848171830177307, -0.4543798267841339, -0.568855345249176, 0.0908694788813591], [-1.009390950202942, 1.9177212715148926, -0.3608816862106323, 0.9572559595108032, 0.7064520716667175, -1.5795272588729858, 0.134871244430542, 1.0170267820358276, -0.9993326663970947, -1.2216532230377197, -0.2034652978181839, 0.5287652015686035, 0.581272304058075, 0.2653510570526123, 0.04175630956888199, -0.16247357428073883, -0.8013140559196472, -0.06814105808734894, -3.373800754547119, 1.655347228050232, -0.8922683000564575, -1.7924081087112427, -0.3541494607925415, -1.963432788848877, -0.760485827922821, 0.4728558361530304, -0.2730758786201477, -1.6021629571914673, 0.450051873922348, -0.8736509084701538, 1.3811368942260742, -0.25594496726989746, 0.06945676356554031, 0.932619571685791, 0.85390704870224, -1.1597214937210083, -1.4916341304779053, 0.6168299317359924, 1.4050759077072144, -0.19643934071063995, 0.5234128832817078, 0.9293972849845886, -0.24345974624156952, -0.8128155469894409, -0.5973725318908691, 1.011813998222351, 1.1757444143295288, 0.3604982793331146, -0.4186381995677948, 0.9384533166885376, -0.9829778075218201, -0.7631296515464783, 0.6484330892562866, 1.9899355173110962, -0.4289051592350006, -0.0810869038105011, -1.1565438508987427, -0.3450040817260742, -0.8240122199058533, 0.2551468312740326, 0.2898246943950653, 1.1257905960083008, 0.08912960439920425, 0.7914147973060608, 0.39293399453163147, 0.4102966785430908, -0.009494924917817116, -0.8890988826751709, 1.7867375612258911, -0.45908087491989136, -0.676283597946167, -1.3602313995361328, -0.46912840008735657, 0.24699725210666656, 1.607358455657959, 0.18157877027988434, -1.0088039636611938, -0.14452572166919708, 1.1155729293823242, 0.6332852840423584, 1.7520761489868164, -0.6534037590026855, -0.2306099385023117, -0.03916275128722191, 0.8124080896377563, -0.5230833888053894, -0.35739752650260925, 0.034238193184137344, -1.4621962308883667, -1.3529328107833862, -0.0005320595228113234, -0.22761370241641998, -2.005403518676758, 2.946507215499878, -0.8314825892448425, 0.4872698187828064, 0.19945749640464783, -0.5311483144760132, -0.3767673075199127, -0.2986603081226349, -0.08227541297674179, -0.3514638841152191, -0.35028526186943054, -0.20100972056388855, -0.9998093843460083, 0.6341805458068848, 1.0930124521255493, -0.06728202849626541, -0.5269211530685425, 0.2225552201271057, 0.5006804466247559, -0.5788846611976624, -0.228017657995224, 0.386831134557724, -0.0008136165561154485, -0.15422900021076202, 0.16031238436698914, 0.5410416126251221, -0.7497059106826782, -0.7757424116134644, -0.19723843038082123, -0.4050130844116211, 0.020282499492168427, -0.2842918634414673, -1.2846670150756836, -1.7655699253082275, -0.9384196400642395, 0.6017132997512817], [0.5763086676597595, 2.081146717071533, -0.002305626170709729, 0.21400724351406097, 0.8143711090087891, -0.9234182834625244, 0.18018923699855804, -0.7299975156784058, 1.0507956743240356, -0.762755274772644, 0.20505589246749878, 0.4107336401939392, 1.227353811264038, -0.1712092012166977, -0.910411536693573, 1.6382203102111816, -0.3970986306667328, 1.111060619354248, -0.176616370677948, 0.12733429670333862, 1.7879469394683838, 1.369896411895752, 0.5667038559913635, -0.28584814071655273, -1.032207727432251, 1.7906558513641357, -0.15248620510101318, 0.823579728603363, -0.01852218434214592, 0.565943717956543, -0.4175027012825012, -0.6487576961517334, -0.07226549088954926, -0.11976782977581024, -1.0307334661483765, 0.4003485441207886, 1.4225438833236694, 0.413556843996048, 0.1086655780673027, -0.8173801898956299, 1.2699205875396729, -0.38543322682380676, -0.2249172031879425, -1.4327412843704224, 1.066861629486084, 1.324324131011963, 0.5936811566352844, -1.2281299829483032, -0.08239822834730148, -1.3280340433120728, 0.41622814536094666, -0.2713965177536011, -0.8403225541114807, 0.3345043659210205, -0.512198805809021, 1.4635040760040283, -1.6150206327438354, 0.37516549229621887, -0.016680698841810226, 0.8139523267745972, 0.7076894640922546, -0.32034653425216675, -1.859468936920166, -1.7586792707443237, -0.5221039652824402, -0.3781348168849945, 1.5957359075546265, 0.16281886398792267, 0.10566501319408417, 1.1186600923538208, 2.180911064147949, -2.132662773132324, -0.4161512851715088, 0.8768884539604187, -1.1690754890441895, -0.903827428817749, 1.1209572553634644, 0.340503066778183, -0.619638979434967, -0.1829933226108551, -0.07763170450925827, -0.47020915150642395, -0.2413797527551651, -0.24314631521701813, -0.49319714307785034, -0.3267577588558197, 0.6679755449295044, 0.2519983649253845, -0.6357901096343994, 1.5336846113204956, -1.1686402559280396, 1.4819936752319336, 0.875217854976654, -1.037040114402771, -1.362058401107788, -1.0464158058166504, 0.7079319953918457, 0.12176477909088135, 0.9653329253196716, -0.6649217009544373, 0.34240415692329407, 0.19477888941764832, -1.424466848373413, -2.174699544906616, 0.3999175429344177, 0.2454690784215927, 0.17807665467262268, -0.0026454743929207325, 1.8522032499313354, -0.1833847016096115, -0.04512523114681244, 0.42638400197029114, 0.009000588208436966, 0.08019997924566269, 2.352771043777466, -0.3291296064853668, -0.20559343695640564, 1.1692819595336914, -2.5886073112487793, -0.41873928904533386, -0.047985516488552094, -0.4779730439186096, -0.016086077317595482, -0.7526049017906189, 0.08344048261642456, 0.6728997826576233, -1.3166838884353638, 0.07950691133737564], [0.6968739628791809, 1.2221808433532715, 0.4154360890388489, -0.4553939998149872, 0.8504927754402161, -2.3895034790039062, 0.7779264450073242, -1.216303825378418, 0.5611506104469299, -0.4294188320636749, 0.137798011302948, 0.4369318187236786, 0.7720382809638977, -1.2847023010253906, 1.1639689207077026, 0.04354735091328621, -0.5684689283370972, 0.7732250094413757, 0.23211081326007843, 0.4391922950744629, 0.2943064868450165, 1.0662801265716553, 1.0605872869491577, 0.43327394127845764, -0.11503922939300537, 1.1778444051742554, -2.3554975986480713, -0.10305216908454895, 0.18495523929595947, 0.9855908155441284, 1.685145616531372, -0.9959137439727783, -1.2095335721969604, -0.018006427213549614, -0.004058086313307285, -1.2876129150390625, -0.540837287902832, -0.6412445306777954, -0.8843070268630981, 0.24196414649486542, 1.2759761810302734, -0.8160830736160278, -0.27336421608924866, -1.068333625793457, -0.18144913017749786, 0.8195898532867432, 0.6750267744064331, 0.6628641486167908, 0.37828198075294495, 0.6055377721786499, -0.25257351994514465, -2.0422542095184326, 0.3210040330886841, 0.8336607217788696, -0.2809606194496155, -0.4729767143726349, 0.9740802049636841, 1.2038397789001465, -0.413094699382782, -0.0760595053434372, -0.3503616452217102, 0.3787578344345093, -1.2885817289352417, -1.324530005455017, -0.10723250359296799, -0.29401302337646484, 0.4750845432281494, 1.3688349723815918, -0.5440806150436401, 0.979855477809906, -1.1913057565689087, -0.26976943016052246, 0.332564115524292, -0.020364560186862946, -0.7629808187484741, -0.9097224473953247, -0.38372939825057983, -0.8810132741928101, 0.20018228888511658, 0.6130644679069519, -0.4281112849712372, 1.350401520729065, -1.349452018737793, -0.35117512941360474, -0.1076846718788147, 0.2357024848461151, -0.8043636083602905, 1.9082437753677368, 0.31264790892601013, -0.7174656987190247, -0.3009724020957947, 1.0912435054779053, 0.13106048107147217, -1.6863312721252441, -0.7070911526679993, 1.2743335962295532, -0.6863323450088501, 1.075183391571045, -0.44623079895973206, 0.49035540223121643, 1.1729629039764404, -0.7259550094604492, 0.653418242931366, 1.7675715684890747, -0.011895732954144478, -0.06893939524888992, 0.4913018047809601, -1.2417365312576294, 1.1363434791564941, 0.8409295678138733, -0.9859721064567566, -0.003408046206459403, -1.0308773517608643, 0.030626345425844193, -0.7658119797706604, 0.19919100403785706, 0.37996989488601685, -0.6866282820701599, 0.639325737953186, 0.9654354453086853, -0.3545372188091278, -0.36675623059272766, -0.47792357206344604, -0.11450134962797165, -0.9162256717681885, -0.2718874514102936, 1.3776512145996094, -1.3356801271438599]], [[-0.783829927444458, -0.6699491739273071, 0.8902294039726257, -0.07000782340765, 1.0039124488830566, 0.42547065019607544, -1.613690972328186, -0.25104692578315735, 1.6043819189071655, 0.3399212062358856, 0.2497270554304123, 1.201418399810791, -0.6913686990737915, 0.29967960715293884, -0.6048392057418823, -0.1478927582502365, 0.19611191749572754, 0.0073818364180624485, 0.56074458360672, 0.25655922293663025, -0.7606425285339355, -0.35001638531684875, -0.26545196771621704, 0.007217556703835726, 1.283019781112671, -0.25888216495513916, -0.026220357045531273, -0.3657929301261902, -0.07256273180246353, -0.9591560363769531, -0.6249162554740906, -1.6280937194824219, 0.26960086822509766, 2.550044059753418, 0.3703993260860443, -0.42431163787841797, 0.035933323204517365, 0.9421348571777344, 0.26723870635032654, -0.36199432611465454, 0.34942537546157837, 0.08025909960269928, -0.6877235174179077, -0.7615296244621277, -0.13374899327754974, -2.146260976791382, -0.0023066725116223097, 1.0866895914077759, 0.26138070225715637, -0.5555503964424133, 0.8321210741996765, -1.0665584802627563, 0.28760480880737305, 1.2630971670150757, -0.7096045613288879, -0.1639525443315506, 1.0895898342132568, 0.8288533091545105, 0.9956247806549072, -0.18169225752353668, 1.2149320840835571, 0.46040210127830505, -1.6295274496078491, -0.4659058153629303, 0.22634363174438477, 0.3120606243610382, -1.8052634000778198, -0.18333357572555542, -2.4682557582855225, 0.3782191574573517, -0.3905681073665619, 0.09701873362064362, 1.096605658531189, 1.6403899192810059, 1.3634575605392456, 0.374778687953949, 0.8573644161224365, 0.5624415278434753, 0.7976146340370178, 0.7444034218788147, -0.013349964283406734, -0.5129961967468262, -1.7735458612442017, 1.9600476026535034, -0.668979823589325, -0.8030834197998047, 0.14897428452968597, 1.51140296459198, -0.43729254603385925, -1.0908937454223633, -1.513734221458435, -0.09002263844013214, -0.02532055787742138, 0.5434502363204956, 0.518565833568573, -0.041504304856061935, -0.3622383773326874, -1.749922275543213, 1.3582960367202759, -0.5730890035629272, 1.5041308403015137, -0.7768863439559937, -0.2014537751674652, -1.00368070602417, 0.2930629253387451, -1.242531180381775, 0.17302505671977997, -1.9498730897903442, 1.4753258228302002, 0.7323043942451477, 1.4072808027267456, -0.9144154191017151, -0.7547871470451355, -0.9817494750022888, -1.0858886241912842, -0.3440057933330536, 0.7467668056488037, 0.3862607181072235, -0.19044898450374603, -0.4772975444793701, 0.7327473163604736, -0.7412974238395691, -0.7843727469444275, -1.962753415107727, -1.7418278455734253, -1.826945185661316, -0.16399995982646942, -0.6774344444274902], [-0.4934457242488861, 2.5194783210754395, 0.15016524493694305, 1.0531398057937622, -0.847176194190979, 0.3073199987411499, -0.05287324637174606, 1.789304494857788, -0.6815351843833923, -0.356868714094162, -1.8245220184326172, 0.9596942067146301, 0.14084357023239136, -2.117366075515747, -0.5126624703407288, -0.22452344000339508, 0.6660440564155579, 0.9706974029541016, -0.44504499435424805, -0.1451282948255539, -0.10294469445943832, 0.7624244093894958, 0.23471999168395996, -0.4448646605014801, 0.686343789100647, -1.7529300451278687, -0.4675178527832031, 0.7248672246932983, 1.7772905826568604, -0.3604591190814972, 0.40353643894195557, -0.21595747768878937, -0.12859059870243073, 0.12901246547698975, -1.1390595436096191, -2.222428560256958, -0.5634883642196655, 0.28218722343444824, 0.5330188274383545, -0.019471392035484314, -0.13000617921352386, -2.4695911407470703, 0.031096970662474632, 0.762319028377533, -0.9979803562164307, 0.49896758794784546, -0.1676894724369049, 1.5650300979614258, 0.5428798794746399, 0.0030422115232795477, 0.28889691829681396, 0.3733527958393097, -0.45885682106018066, -0.23530572652816772, 0.830369234085083, 0.9506152868270874, -0.2525882124900818, -1.1348947286605835, -2.1659889221191406, 0.7877999544143677, -0.632259726524353, -0.7692251801490784, -1.3674763441085815, 0.6403077244758606, 1.9542746543884277, -1.4765299558639526, 0.9024468064308167, -1.769095540046692, -0.9555574059486389, -0.9986515045166016, -0.8584395051002502, -0.23622702062129974, -0.1990777999162674, -0.9950893521308899, -0.4852046072483063, -1.117372989654541, 0.8495969176292419, 1.6299432516098022, 0.06124820560216904, -0.8714560270309448, -0.3964475095272064, 0.3608298897743225, -0.5641316771507263, 1.0742626190185547, 0.6575286388397217, -0.45259925723075867, 1.018876552581787, 0.2560640275478363, 1.5992811918258667, -0.20558024942874908, 0.9500181078910828, 0.609281599521637, -1.0103083848953247, 0.6065719127655029, 0.281229704618454, -1.3674060106277466, -1.8074798583984375, -1.633097529411316, 0.8913966417312622, -1.2654125690460205, -1.1807953119277954, -0.13014325499534607, -0.5526072382926941, 0.1563921719789505, 1.1820783615112305, 0.1569642573595047, -0.18442794680595398, 1.7105605602264404, 0.39972150325775146, 2.177337884902954, 1.8781567811965942, -0.24984519183635712, -0.29688504338264465, 0.8080618977546692, 1.711159348487854, -0.12743046879768372, -0.4777509272098541, 0.7528270483016968, 0.2440406084060669, 0.6421346068382263, -0.2965739667415619, -1.01736319065094, -1.7214841842651367, -2.2237777709960938, 0.005262542515993118, -0.7248466610908508, -0.01544992532581091, -0.2795834243297577], [-0.6323152780532837, -0.5643066167831421, 2.463148355484009, -0.5675698518753052, -0.46649518609046936, -0.4498867690563202, 0.770984411239624, 0.3545266091823578, 0.11140631139278412, 0.9392080307006836, -0.984076738357544, -0.37558525800704956, 0.587222158908844, 0.18978412449359894, -0.7003217935562134, -0.5840687155723572, 1.710982084274292, 0.278085321187973, -0.04503077641129494, 0.673328161239624, -1.1823248863220215, -0.4309571087360382, 0.29765936732292175, 0.5430475473403931, -0.20932415127754211, 1.2750656604766846, -3.5466325283050537, 0.3240337371826172, 0.8515687584877014, -0.19474905729293823, -0.6169548630714417, -0.39771568775177, -0.4744527041912079, 0.7522315979003906, 0.2804078459739685, 1.5144128799438477, -0.6933291554450989, 0.1400422602891922, -1.5409839153289795, 1.0440176725387573, 0.5602430105209351, -0.061061907559633255, 0.18562380969524384, 0.23935581743717194, -0.45856326818466187, -1.164552092552185, 0.6399939060211182, 1.5912914276123047, -0.6700191497802734, 1.3061347007751465, -0.7404232621192932, -1.3512458801269531, -0.5792381167411804, -0.8665628433227539, -0.23914356529712677, 0.46055641770362854, 0.027379287406802177, 0.025117116048932076, -0.6488712430000305, -1.0294438600540161, 0.6873692870140076, 0.2707534432411194, -0.35619693994522095, -0.8356716632843018, 2.0929503440856934, -1.4114015102386475, 0.1159788966178894, 0.14497266709804535, 1.7151157855987549, 1.0621590614318848, -0.894227147102356, -1.4664674997329712, 1.1108038425445557, -1.3851858377456665, 0.6643754839897156, -1.0109301805496216, 0.17591461539268494, -1.591963768005371, -0.2742287218570709, 1.9880536794662476, 1.57386314868927, -0.11622651666402817, 1.487572193145752, 0.4911497235298157, -1.3000346422195435, -0.6360448598861694, -2.026500701904297, 0.6210840940475464, -0.9902346134185791, -0.9013968110084534, -1.4270437955856323, 2.4291868209838867, -0.4938393533229828, 2.057197332382202, -0.3594268560409546, 2.7899534702301025, -1.209814190864563, 0.5048532485961914, 0.997687578201294, 1.3742612600326538, 1.6614820957183838, -0.5279254913330078, -0.729452908039093, -0.9026844501495361, 0.5289773344993591, -1.3553662300109863, -0.3524569869041443, 0.7775131464004517, -0.4175679087638855, -0.11005880683660507, 1.1535344123840332, 1.0183818340301514, -1.160442590713501, 0.4902423322200775, 0.019692106172442436, 1.0609841346740723, -0.8196881413459778, 0.5484899878501892, -0.4171235263347626, 0.2976877987384796, -0.7642583250999451, 1.6179859638214111, -0.2844104766845703, 0.6231817603111267, -0.9385481476783752, -1.1213418245315552, -0.8258628249168396, -0.602936327457428], [-0.11095781624317169, -0.2571030557155609, -0.4047876298427582, 1.5859694480895996, 0.27882951498031616, -0.005149048753082752, -0.9060094952583313, -0.5257459878921509, -1.2680399417877197, -2.140763998031616, 0.7495003938674927, 1.537514567375183, -0.25917714834213257, -0.17197853326797485, 1.3963528871536255, 0.750033438205719, 0.5712385773658752, -0.006336594466120005, -1.3493537902832031, -0.17362354695796967, -0.6333430409431458, -1.876781702041626, 1.9438809156417847, 1.2166677713394165, -0.18232716619968414, 0.46752429008483887, -0.7634376287460327, -0.4877503514289856, -0.6154467463493347, 1.2189189195632935, -0.29148420691490173, -0.78349769115448, -0.5518575310707092, -0.5887243747711182, 1.2973288297653198, -0.10051143169403076, 2.5366063117980957, 1.1098697185516357, 0.554206132888794, 0.31298258900642395, -0.7778330445289612, 1.14589524269104, 1.0820928812026978, 0.19568109512329102, -0.6609970927238464, -1.543216586112976, 1.4911057949066162, 1.8168913125991821, 1.7013789415359497, 1.33324134349823, -0.5531721711158752, 0.17532648146152496, -0.5757947564125061, -0.7966152429580688, 1.438712477684021, -0.6461631655693054, 0.03541124612092972, -1.0276752710342407, 0.45026835799217224, -0.26045358180999756, 0.4131101965904236, 0.8088595867156982, -0.6750319600105286, 1.7092947959899902, 1.9352790117263794, 1.1248204708099365, -1.6068240404129028, 1.3696262836456299, -1.9313652515411377, 1.1618657112121582, 0.4576318562030792, 0.6309959292411804, -1.7402127981185913, 0.025349527597427368, -1.2979761362075806, -0.4697130620479584, 0.06636709719896317, 1.006467342376709, -0.39791345596313477, 1.84469473361969, -0.6281697750091553, -0.5476555824279785, 0.8660798668861389, 0.5671270489692688, 0.8072747588157654, 0.24637340009212494, -0.8780646920204163, -1.471300721168518, 1.348764419555664, -1.2755134105682373, -0.3957444727420807, -1.1323442459106445, -0.24962107837200165, -0.4493982493877411, -1.5617825984954834, -0.5810452103614807, -1.2916404008865356, 0.43224963545799255, 0.4350689649581909, 0.43570929765701294, 0.636674165725708, 1.032160997390747, 0.3417280912399292, -0.8965650796890259, -0.3087373971939087, -0.23659217357635498, -0.9183416962623596, 0.9085791110992432, -0.8442639708518982, -0.7605013251304626, -0.20355914533138275, -1.401365041732788, -0.9285013675689697, -1.0484983921051025, -0.8434883952140808, -0.8354001641273499, 1.0846093893051147, -0.5360236167907715, 0.13793890178203583, -0.7151302695274353, 0.21877306699752808, -1.2808386087417603, -0.6041355729103088, -2.849501132965088, -0.6736730933189392, -0.8044220209121704, 0.08391494303941727, -0.814704954624176], [-0.8427391052246094, 2.3176846504211426, 1.495679497718811, -0.33588090538978577, -0.7185012102127075, -0.32831889390945435, -0.9954744577407837, -0.9221412539482117, 0.10117282718420029, 0.7456324100494385, 1.6155061721801758, -2.1835720539093018, -0.3078606128692627, -0.5459253191947937, -0.2787478566169739, -0.008335855789482594, 1.2434316873550415, -0.23145005106925964, 0.2687374949455261, -0.2868548333644867, 0.676002025604248, 1.4622119665145874, 0.7165082693099976, 0.7712153792381287, -1.4784960746765137, 1.016833782196045, -0.2621106505393982, 0.10167568176984787, -0.5751744508743286, 1.1896296739578247, -0.23583127558231354, 0.75334233045578, 0.6287490725517273, 0.26634085178375244, 1.1543192863464355, -0.25574469566345215, 1.2370755672454834, 0.640067994594574, -0.4392206370830536, -1.0225257873535156, -0.4246237874031067, -1.976402997970581, -0.5255288481712341, 1.4799861907958984, 1.6161688566207886, -0.3474765121936798, -1.4231830835342407, 0.4031751751899719, -0.19604647159576416, 1.103330373764038, -1.3719301223754883, 1.200135350227356, 1.505537748336792, 0.35877931118011475, -1.1993390321731567, -1.0789142847061157, -0.07638653367757797, -0.5524026155471802, 0.6955615282058716, -0.38423770666122437, 0.9045871496200562, 1.315445899963379, -0.417015939950943, -0.5273891091346741, 0.7849816679954529, -0.04965026676654816, -0.5108321309089661, -0.518927812576294, 0.732189416885376, 0.9954941272735596, -1.0724949836730957, -0.226702019572258, -0.39387622475624084, -0.12011582404375076, 1.1526507139205933, 0.478491872549057, 0.533713161945343, 1.1156007051467896, 0.17297376692295074, 0.6201508641242981, 1.0060410499572754, 0.9711675047874451, 0.4112713634967804, 0.7853886485099792, 2.953662872314453, -0.6371814608573914, 0.4901692569255829, 1.4747916460037231, 0.3842867314815521, 0.02172093093395233, -0.2870345413684845, 1.029006838798523, -0.03476522117853165, -0.9570298194885254, 0.48406219482421875, -0.2162783443927765, 0.14705795049667358, 0.2132178544998169, -1.4254359006881714, -0.8707214593887329, 0.12446340918540955, 2.4044859409332275, 1.5026466846466064, -0.4126266539096832, -1.0589497089385986, 1.3677022457122803, 1.008388638496399, -0.01620817743241787, -1.1621471643447876, -0.4743008315563202, 0.18092823028564453, -1.0779235363006592, -0.9075405597686768, -0.3387173116207123, 0.50214684009552, 0.16481617093086243, 0.2655441462993622, -0.3982042968273163, 1.775809645652771, 0.9060655236244202, -0.0540505088865757, -0.12045391649007797, 0.2533249855041504, -0.7724977135658264, -0.8121533393859863, 0.7651538252830505, 0.15281827747821808, 1.6373132467269897], [0.15053145587444305, 0.9304575324058533, -0.384578675031662, -0.007367467973381281, 1.5291823148727417, 2.006168842315674, 0.650174617767334, -1.0451297760009766, 1.4153655767440796, 0.6655828952789307, 0.4385821223258972, 0.05299137532711029, -0.1473310887813568, -0.994932234287262, -1.4384360313415527, -3.1190075874328613, -0.2252018004655838, -0.4004628658294678, -0.7592173218727112, -0.6159814596176147, 0.4269867539405823, -0.6627020835876465, -0.07285512238740921, 0.9340572357177734, 0.8248418569564819, 0.33478617668151855, -0.24994277954101562, -1.0571311712265015, -0.4502105712890625, -0.5964949727058411, 0.3223753273487091, -0.23082466423511505, 0.9868380427360535, -1.0217615365982056, -0.4417850375175476, 1.0093873739242554, -1.2848501205444336, -1.269415020942688, -1.679500937461853, -1.432144284248352, 0.29687345027923584, -1.2299513816833496, -1.0226808786392212, -1.0762097835540771, 1.4282833337783813, 2.711764335632324, 1.5677131414413452, 0.8031371831893921, -0.3733691871166229, -0.6423697471618652, -0.9447156190872192, 1.472653865814209, -2.153801679611206, -0.6387927532196045, -0.5212746858596802, 0.4784979522228241, -1.6575027704238892, -0.0007630400359630585, -0.7371842265129089, 0.7819066643714905, 0.7884593605995178, 0.3683070242404938, 0.8542937636375427, 1.257912039756775, 0.9563064575195312, -1.5504435300827026, -1.997555136680603, -0.05390524864196777, 2.1054000854492188, 2.4056458473205566, -0.6966675519943237, -1.4320993423461914, -0.611469030380249, 0.5827867388725281, -0.8432411551475525, 0.9465194344520569, 1.2839473485946655, -0.08707716315984726, -0.1953684687614441, 0.8607439994812012, 0.7654643058776855, -0.031261228024959564, 0.3131684958934784, -1.5294498205184937, 2.7019591331481934, 0.11374101787805557, -1.1849746704101562, 0.7781582474708557, -0.6787111759185791, -0.15043699741363525, 1.5669749975204468, 0.14074033498764038, -0.4027341306209564, -0.7281877398490906, -0.3114010691642761, -0.4599674344062805, 0.13568903505802155, -1.2331093549728394, -0.5799238681793213, -0.5165960788726807, 0.7169005870819092, -0.0963490754365921, -0.5402697324752808, -1.2886320352554321, 0.613208532333374, 0.03196243941783905, -1.404940128326416, 0.022539813071489334, -0.2678877115249634, -1.6616209745407104, -0.40382876992225647, 1.5014067888259888, -0.5921662449836731, 1.135225534439087, 0.4648467004299164, -0.6513588428497314, 1.1906065940856934, 0.06873651593923569, 0.1181112751364708, -1.3569207191467285, -0.623245894908905, -1.1401066780090332, 0.6458588242530823, -0.8178001046180725, -1.0215898752212524, 0.6013060212135315, -0.8223710656166077, -1.014479637145996], [0.5087855458259583, -0.3565680980682373, -1.9081956148147583, 0.8798692226409912, -0.8165145516395569, 1.9846818447113037, -0.5113517642021179, -1.280795931816101, 1.0910042524337769, -0.7449092864990234, -0.613734245300293, 1.7602170705795288, 0.3677506446838379, -0.330753356218338, 1.2863212823867798, 0.06994892656803131, 1.0446176528930664, -0.031232908368110657, 0.23147797584533691, -0.8319517970085144, -0.4888160526752472, -0.591232419013977, 0.6883261799812317, 0.38012826442718506, -0.17117169499397278, 1.4480823278427124, 0.2375091314315796, -1.1409820318222046, 0.5632595419883728, 0.729250431060791, 1.143095850944519, -0.010269759222865105, 0.6134427785873413, 1.5999969244003296, -0.38163942098617554, 2.257784366607666, 0.8875465393066406, 2.099238395690918, -0.3947252333164215, 0.17330145835876465, -0.7198073863983154, -0.7760651111602783, 0.20769882202148438, -0.5901899933815002, -1.1631792783737183, -2.345099449157715, 0.022501369938254356, -1.263706922531128, -0.4460015892982483, 0.3963383734226227, -2.2125749588012695, 0.9749841690063477, -0.1340314894914627, 0.010065197944641113, 0.9868162274360657, -0.8809483647346497, 1.595190405845642, 1.1194013357162476, -0.4483692944049835, -0.09358025342226028, 0.6255583763122559, 0.33008989691734314, 0.8078299760818481, -0.6645509004592896, -0.9805921912193298, -0.0875156819820404, 0.23600485920906067, -0.39459046721458435, 1.36549711227417, -0.8959894180297852, 0.8164353966712952, 1.122449517250061, 0.9426388144493103, 0.463095098733902, -0.6103013753890991, -0.7199363112449646, 1.728883147239685, -0.5505810379981995, 1.5259554386138916, -0.6450443267822266, 0.016518739983439445, 0.37398484349250793, 1.8744937181472778, 0.03218473866581917, -1.6206331253051758, 0.5020762085914612, 0.6018590331077576, -0.6684430837631226, 0.587135910987854, -2.6101033687591553, -0.04867630451917648, 0.19602760672569275, -0.8456482291221619, -1.4795578718185425, -1.7429757118225098, -0.5988441109657288, -0.266830176115036, 1.4396296739578247, 0.8331855535507202, -0.786307692527771, -0.13801684975624084, 0.863386332988739, -1.132095217704773, -0.8800965547561646, 0.38612428307533264, 0.8668399453163147, 0.29838263988494873, -0.277545690536499, -0.5635042786598206, 0.48231297731399536, -1.2254939079284668, -0.5445601344108582, -0.4950577914714813, 0.15490348637104034, 0.7924984693527222, -0.1276961863040924, 0.41640704870224, -1.3471680879592896, -0.7452226877212524, 1.057694435119629, -0.7740607857704163, 0.18979467451572418, -0.13707159459590912, -0.11946873366832733, -1.3376425504684448, 0.3308563530445099, 0.5897204875946045, 2.389309883117676], [-0.6865536570549011, 1.265928030014038, 0.8965938091278076, -0.27378711104393005, 2.0328307151794434, 0.45057687163352966, -0.19379335641860962, 0.4260920584201813, 0.4197944402694702, -0.6386586427688599, -0.7893370985984802, -0.2119634747505188, 1.6688580513000488, 0.8754501342773438, 0.4495597779750824, 0.1041162833571434, -0.8585598468780518, 1.1711699962615967, 1.4645293951034546, -0.42763909697532654, 0.05901283398270607, -0.5381608009338379, 0.3664151430130005, 1.2595245838165283, -0.5870397090911865, 0.6677567362785339, -1.5949647426605225, 0.41824957728385925, -1.0936203002929688, 0.2590384781360626, -0.3240397572517395, 0.6085789203643799, -1.600229024887085, -1.1643706560134888, -0.25114139914512634, 0.5891391634941101, 1.5365523099899292, -1.4016557931900024, -2.4180660247802734, -0.3165042996406555, 1.16932213306427, 0.11203933507204056, 0.7875513434410095, -0.8860413432121277, 1.4029700756072998, -2.111210584640503, -0.7715858221054077, -2.8536272048950195, -1.9782274961471558, -0.31646397709846497, 0.3554675579071045, -0.687561571598053, -0.3605266213417053, 0.44870615005493164, -0.3044907748699188, -0.3373083174228668, 0.8277751207351685, -0.5248220562934875, -0.8275430202484131, 0.3517323136329651, -2.2713096141815186, 0.3350382447242737, 0.6370731592178345, -0.2511066198348999, 2.4073898792266846, 0.6001157164573669, -0.9628444314002991, -1.2814624309539795, -1.1547633409500122, -1.090219259262085, -0.37075385451316833, 0.9780303239822388, -0.5176101326942444, 1.8680719137191772, -0.48844820261001587, -1.4046169519424438, -0.5122953057289124, 0.6844784021377563, -1.1313647031784058, 0.14067065715789795, 0.4874141216278076, 0.3183024227619171, -1.4564827680587769, 0.520081102848053, 1.186286211013794, -0.12975040078163147, -0.7730720639228821, -0.167343869805336, 0.39000192284584045, -1.5853745937347412, 0.055042605847120285, 0.3717549741268158, 1.7435506582260132, -0.15543681383132935, 0.294831782579422, 0.40516868233680725, -1.2718595266342163, -0.3966929316520691, 0.004678498022258282, 1.8892425298690796, 1.1282235383987427, 1.7869935035705566, -0.6721200942993164, 1.4276350736618042, 0.1746438890695572, -0.08576696366071701, 0.8381829261779785, 0.43995726108551025, 0.4954344630241394, -1.1133854389190674, 0.38916507363319397, 0.4562324583530426, 0.05105608329176903, -1.5225071907043457, 2.17743182182312, 1.1219562292099, -1.2329668998718262, 0.4204833209514618, 1.1538960933685303, 0.32842662930488586, -1.0390329360961914, -0.6470673084259033, -0.3993928134441376, 1.0280109643936157, -0.32096585631370544, -0.45836231112480164, -0.5461694598197937, 0.29042091965675354]]]], "dtype": "float32"}, {"__nd__": [[[[0.14756222069263458, -1.756488561630249, -1.998317003250122, -1.5183323621749878, 0.5859243869781494, 1.6134583950042725, 0.43016910552978516, 0.3407115340232849, -0.8089923858642578, -2.0417869091033936, 1.1442577838897705, -0.3572036921977997, -0.6574452519416809, -0.8213720917701721, 0.4663980305194855, -0.1453951746225357, -1.6439787149429321, 1.0158125162124634, 0.5166407227516174, -0.1265036016702652, -1.0899252891540527, -1.1511199474334717, 0.8186179995536804, -0.3307734727859497, -0.7120373845100403, -0.5576993823051453, -1.05972421169281, -0.00929571408778429, -0.549187421798706, 1.4532824754714966, 0.24464458227157593, -0.6309264898300171, 0.265250027179718, 2.780128240585327, 0.7097545862197876, -0.30598336458206177, 0.4238748550415039, 0.7568396925926208, 0.15804535150527954, -1.0974557399749756, -1.1014611721038818, 1.0536943674087524, 1.6835181713104248, -0.5084490776062012, 0.6874261498451233, -0.38517287373542786, -0.9207248687744141, -0.5222784876823425, 0.6804473400115967, -0.14657093584537506, 1.6797173023223877, 0.051642537117004395, 0.251486212015152, 1.1258838176727295, 0.8373645544052124, 0.4350598156452179, -1.5344021320343018, -0.7424403429031372, -0.5115469694137573, 1.2337158918380737, 1.9590039253234863, -0.269197016954422, -1.6601389646530151, 0.8790742754936218, 0.6385685205459595, 0.08385740220546722, -0.059855975210666656, 0.5803107619285583, 0.33514299988746643, 1.4372413158416748, -0.24910376965999603, 1.1383185386657715, 0.04971575364470482, 0.22599245607852936, -1.399324893951416, 1.3275266885757446, -0.2645282447338104, 2.1456058025360107, -0.04050830379128456, 0.8246050477027893, 0.5944820046424866, 0.7414475083351135, 0.14113812148571014, 1.219852328300476, 0.28645387291908264, -1.0633383989334106, 1.0491400957107544, -0.5618233680725098, 0.08243533968925476, 2.140413522720337, -1.8659133911132812, -0.6032038927078247, -0.439558207988739, -1.1292626857757568, -1.0316987037658691, 0.3508196473121643, 0.4653181731700897, 0.6281521916389465, -0.7895999550819397, -1.5276058912277222, -0.30645954608917236, -0.18614356219768524, 0.7699382901191711, -0.44809988141059875, -0.47059470415115356, -0.5603837370872498, -1.5963859558105469, -0.4420664310455322, 1.2615076303482056, 1.2843267917633057, 0.1926543116569519, 0.500058650970459, -0.38413992524147034, -0.3487975597381592, -0.44347864389419556, -0.2837411165237427, 0.6646482944488525, -0.48662734031677246, 1.1156796216964722, 1.9734854698181152, -0.2521001696586609, -0.8118206262588501, -0.32882654666900635, 0.6934378743171692, -1.287653923034668, 1.070525050163269, -0.06432486325502396, -0.20067769289016724], [-1.8266206979751587, 0.2736470699310303, 0.1974164843559265, 0.07289745658636093, 0.2199956774711609, -0.5784772634506226, -0.7653297781944275, -0.3786073625087738, 1.3794169425964355, 2.1065633296966553, -0.8351499438285828, -0.9351839423179626, -1.1139423847198486, 1.3339383602142334, 1.006820559501648, 0.4944870173931122, -0.5127921104431152, -0.3523416221141815, 0.7857489585876465, 0.3533872365951538, -0.1868741363286972, 0.08996418863534927, -1.6862682104110718, -0.1092061772942543, -0.365820050239563, -1.2299561500549316, 0.33194589614868164, 3.0679070949554443, 1.6862847805023193, 0.6507308483123779, -1.5062919855117798, 0.0008892926853150129, 1.252263069152832, -1.6585091352462769, 1.793784499168396, 1.5513356924057007, -0.5518672466278076, 0.20021355152130127, -0.7173019051551819, -0.9067236185073853, -0.8182609677314758, 1.2808743715286255, -1.6999279260635376, -0.05118729919195175, -0.10257872939109802, 0.14407862722873688, -0.940605878829956, 1.1451160907745361, -1.4863567352294922, -0.9599628448486328, -0.02324517071247101, 0.195995032787323, 0.7537611126899719, 0.14879892766475677, 0.5586304068565369, 0.259619802236557, 0.14371521770954132, -0.09998616576194763, -0.8468367457389832, -1.3809494972229004, -1.9962782859802246, 0.3079468607902527, 0.00014981644926592708, 0.43732696771621704, 0.09317848831415176, -1.3287187814712524, -0.13884702324867249, 0.44899943470954895, 0.1952906847000122, -0.8259063363075256, 1.1041603088378906, 1.552138328552246, -0.1366511583328247, -1.958500623703003, -0.8653796911239624, 1.0135526657104492, -1.8240160942077637, -0.6251847147941589, -1.591975450515747, -0.20957808196544647, -1.4616276025772095, -0.37047508358955383, 0.8899039626121521, -1.3073428869247437, 0.6325360536575317, 0.07947264611721039, -1.4170645475387573, 0.6012691259384155, 0.6026713848114014, -0.16142958402633667, 0.5207289457321167, -1.2009937763214111, -1.581918478012085, 0.38961857557296753, 1.67043936252594, 1.2842472791671753, 0.7217782139778137, 0.7175482511520386, -0.6713602542877197, 0.31736835837364197, -0.7702494859695435, -1.0634950399398804, 1.4951941967010498, -1.131585717201233, 1.5878281593322754, 0.3362096846103668, 0.6345281004905701, -1.2743406295776367, 0.23083829879760742, -0.8096376657485962, -0.2905203402042389, -0.8753916621208191, 0.5826163291931152, 0.3329046368598938, -0.398441880941391, -0.263047456741333, -1.715826392173767, -0.3852766752243042, 0.6293668746948242, 1.4052776098251343, 0.22062039375305176, 1.1300731897354126, -0.07500334829092026, -1.8794963359832764, -0.02011132799088955, 0.08359727263450623, 1.1767112016677856, -1.28414785861969], [2.2584457397460938, 0.719716489315033, 1.545499324798584, 1.4400880336761475, 1.7616405487060547, -1.2931430339813232, 0.3044625222682953, 0.3434430658817291, 1.1326707601547241, -1.5744011402130127, 0.6126816272735596, -0.751674234867096, -0.5723467469215393, 0.12641222774982452, -0.46349257230758667, 0.6712644100189209, -0.5040537118911743, 0.4979339838027954, -0.7214226722717285, -0.9390572905540466, 1.0711185932159424, -0.19771263003349304, -1.058450698852539, -0.18281692266464233, 0.19811351597309113, -0.2514105439186096, -0.9492900967597961, -1.3713829517364502, 1.6246004104614258, 0.48190057277679443, 0.20630861818790436, 1.7213184833526611, -1.3517965078353882, -1.6237653493881226, -1.3583440780639648, 1.2153213024139404, -1.6891717910766602, -1.8717756271362305, -0.9307138919830322, -1.0907584428787231, 0.44724100828170776, 0.22637884318828583, 0.8903431296348572, 0.26713326573371887, 0.09127204865217209, 0.30070239305496216, -1.8768084049224854, -0.2954253852367401, 1.2389837503433228, -0.28554224967956543, -1.243493914604187, 0.14831092953681946, -0.24031750857830048, 0.16652843356132507, 0.8738677501678467, -0.5355395674705505, 0.9699069261550903, 0.5723503828048706, -1.0268454551696777, 0.6222113966941833, -0.919651985168457, -0.9028031229972839, 0.0846688374876976, 0.5498903393745422, 0.5064373016357422, 0.33786725997924805, -0.8314858078956604, -1.1785908937454224, 0.2542344629764557, -1.3644739389419556, -0.3991023600101471, 0.08103025704622269, 1.8610965013504028, 0.022994382306933403, 0.31265154480934143, 0.8585863709449768, 0.2501020133495331, 0.9139604568481445, 0.23337148129940033, 0.8943371772766113, 0.3536342680454254, -0.4822513163089752, -1.31620454788208, -0.09113487601280212, -1.025866150856018, -1.1885993480682373, -0.34851109981536865, -1.812669277191162, -0.3094046413898468, 0.5648342370986938, -0.5474462509155273, -0.3515218496322632, -0.46617963910102844, -0.06945651769638062, 2.283256769180298, 0.7937182784080505, 0.6306225657463074, -1.5507378578186035, 0.45123741030693054, 1.1045857667922974, -1.205280065536499, -1.7465696334838867, -0.10737957805395126, 0.22623282670974731, -1.2033015489578247, -0.49172258377075195, -1.5212973356246948, -0.6000126600265503, -1.6631439924240112, -1.3476314544677734, 1.2625831365585327, 1.9894121885299683, 0.7114508152008057, 1.6562548875808716, -1.568019986152649, 0.5999391674995422, 0.7315685153007507, 0.7910407781600952, -0.40738028287887573, 0.19438882172107697, -0.5667858719825745, 1.2056457996368408, -1.0662240982055664, 0.04220876097679138, -0.5875095129013062, -0.16535310447216034, 1.0585232973098755, -1.5075957775115967], [-0.8658073544502258, -0.14907285571098328, -0.43631190061569214, 0.16276532411575317, 1.0294888019561768, -0.45318111777305603, -0.5197022557258606, 1.408582091331482, 0.22001108527183533, -2.024521827697754, 1.101718544960022, 0.6318673491477966, 0.7780770659446716, 0.02488042786717415, 0.30743110179901123, 0.21579059958457947, -0.9851107597351074, 1.4099725484848022, 1.2207547426223755, -1.0391370058059692, -0.3532388210296631, 0.6198177933692932, -0.8752499222755432, 0.09258566051721573, 1.2687073945999146, -1.2288058996200562, 1.1802210807800293, -0.09466078877449036, 0.8676087856292725, -0.8265783190727234, -1.7259225845336914, -0.1837199330329895, 0.10590910911560059, 0.8057875037193298, -1.2691997289657593, -0.590634822845459, 0.8543539047241211, 0.7464165687561035, 1.0477111339569092, -0.1158503070473671, 2.1316587924957275, 0.9357845783233643, -0.0037085069343447685, -0.21345943212509155, -0.5859249830245972, -1.6570252180099487, -0.7311909794807434, -0.8436319828033447, 0.5681130290031433, -0.41867873072624207, 1.0071756839752197, 0.6158161759376526, 0.6905049681663513, 0.47829508781433105, -0.8060135245323181, 1.3808178901672363, -1.2687361240386963, 1.1741876602172852, 1.3669949769973755, 0.910146176815033, 0.7750175595283508, 0.5354381799697876, -0.40778428316116333, -0.20986537635326385, 0.028381304815411568, 0.19543810188770294, 0.7282541990280151, 0.619904100894928, -0.5993713736534119, -1.4733952283859253, -0.7137997150421143, -1.544934630393982, 0.867165744304657, -0.1949915885925293, 0.06377548724412918, -1.2628895044326782, 0.6368908882141113, 1.6053941249847412, 1.564163327217102, 0.6231932640075684, 0.794434666633606, -0.6726589202880859, -0.05583646520972252, -0.163417249917984, 0.9433929920196533, 0.33177414536476135, -1.3944716453552246, 0.709028959274292, 0.08126184344291687, -1.2586122751235962, -1.0074331760406494, -0.004425701219588518, 1.1995865106582642, 0.5982969999313354, 0.25007542967796326, -0.41699013113975525, -0.06837523728609085, 1.0050742626190186, -0.8611849546432495, -1.425872564315796, 0.29291337728500366, 1.1866724491119385, 1.6624170541763306, 0.38555651903152466, -0.553127110004425, 0.2158263623714447, -1.062371015548706, -1.1209381818771362, -0.07332668453454971, 0.46772435307502747, 0.3621155321598053, 0.3661459982395172, 0.1877431571483612, -0.45608341693878174, -1.3457515239715576, -0.17081904411315918, -1.6308492422103882, -1.3118460178375244, -0.7191922068595886, -0.3210601806640625, 0.502600908279419, 0.7560752034187317, -0.2977782189846039, 0.2718418538570404, 0.1824020892381668, 1.5930702686309814, 0.05200079083442688, -1.1145356893539429], [-0.9104647636413574, -0.5297331809997559, 0.7767089009284973, -0.34060025215148926, -0.1739197075366974, -1.3998326063156128, -1.6859452724456787, -2.1531014442443848, -2.1298351287841797, 0.29650959372520447, -0.3658612072467804, 0.9947973489761353, -0.5479015707969666, 0.0709971934556961, -0.44172272086143494, -0.8561437129974365, 0.8103192448616028, 0.6553829908370972, -0.29796943068504333, -0.36919811367988586, 0.9661928415298462, 0.5078933238983154, 0.0005855046911165118, 0.41718170046806335, -0.5845571160316467, 1.1954036951065063, 0.7909828424453735, -0.48398998379707336, -0.5728192925453186, 0.022903723642230034, -1.1551673412322998, 0.8619862794876099, -0.08194328844547272, 1.0440236330032349, 0.364721417427063, -1.358120083808899, 0.8022478818893433, 0.44430872797966003, -0.4522494375705719, -1.1170798540115356, -1.1071990728378296, 0.006908607669174671, 0.10034039616584778, 1.0471889972686768, 1.4334298372268677, -1.452658772468567, -2.369534730911255, -0.23515786230564117, 0.6554028391838074, -0.9045606255531311, -0.44115281105041504, -0.23144720494747162, -0.1660090833902359, -0.49486005306243896, 0.2718406915664673, 0.7324798703193665, -0.5744054913520813, 0.07629416137933731, 1.1442487239837646, 2.129568099975586, -1.3258899450302124, 1.0729576349258423, 0.43842899799346924, -0.5270740985870361, -0.8712998628616333, 0.14068563282489777, -1.3158437013626099, 1.4755345582962036, 0.6845932602882385, 1.402442455291748, 1.515106439590454, -0.4578952491283417, 1.2143034934997559, -0.48313960433006287, -1.341995120048523, 0.1157468631863594, 0.4500335156917572, 0.279207706451416, -0.7163474559783936, -2.1681933403015137, 1.2425010204315186, 0.8252659440040588, 0.4232892096042633, 1.0285279750823975, 0.9786748290061951, -2.917762041091919, -0.37847182154655457, -0.9911680221557617, -0.5787078738212585, -0.5752539038658142, 0.8232820630073547, 1.3518881797790527, 0.3688737154006958, 1.3870387077331543, -0.03822529688477516, -0.9857311844825745, -0.545979917049408, -1.860626220703125, -0.700087308883667, -0.9193523526191711, -0.7693521976470947, 0.43070027232170105, -0.31587621569633484, -1.1569288969039917, -1.3432872295379639, 0.11343150585889816, 0.9117504358291626, -1.1108119487762451, 0.31941235065460205, 1.251198649406433, 2.3038241863250732, -0.4502599537372589, 0.9997724294662476, -0.6438467502593994, 0.8417339324951172, 0.47112083435058594, -0.7408431172370911, 1.0158674716949463, -0.5342807769775391, 0.6624866724014282, 0.36125674843788147, 0.07490154355764389, 0.8478748202323914, -1.77129065990448, 0.24427253007888794, 1.5793639421463013, -1.321338176727295, 1.2520142793655396], [0.9811595678329468, -1.4196833372116089, -0.6837161183357239, 0.46958979964256287, 0.044723883271217346, 0.062406644225120544, 1.1580561399459839, 0.6443729400634766, -0.3885575830936432, -0.7183971405029297, -0.43438366055488586, 0.7125962376594543, 1.663394808769226, -0.8120887279510498, 0.48418110609054565, 0.44212082028388977, -0.027944523841142654, 0.4871971011161804, -0.016405774280428886, 0.7025940418243408, 1.0798622369766235, -0.25015273690223694, -1.0711593627929688, -1.0797209739685059, -0.8466324210166931, -1.218333125114441, -1.2194839715957642, -0.007891084998846054, -0.06312929093837738, -1.5888413190841675, -1.2392557859420776, 1.0005083084106445, 0.49214908480644226, 0.06672839820384979, 0.8640691041946411, -1.1503795385360718, -0.30739012360572815, 0.7510876655578613, -0.4927622079849243, -0.2320139855146408, -1.1058037281036377, -1.0373154878616333, 0.1928325742483139, 0.33318030834198, -1.0070332288742065, -0.48812395334243774, -1.8974640369415283, -0.9452022910118103, 0.17610754072666168, 0.30095139145851135, -1.0559916496276855, 0.37692978978157043, 1.6889601945877075, 0.3553306460380554, 0.7970554232597351, 0.2821429967880249, -2.368090867996216, -0.2629470229148865, -0.6623579859733582, -0.5418910980224609, -1.1374092102050781, 0.9459249377250671, -0.37441202998161316, -0.4824778735637665, 1.804599642753601, -0.35354122519493103, -0.4680604934692383, -0.6391445994377136, -0.271132230758667, -3.030207872390747, 0.02121635712683201, 0.5473207831382751, 0.15225467085838318, 1.472144365310669, 0.35478994250297546, -0.5910152196884155, 0.4664137363433838, -0.9987773895263672, 0.7435832619667053, 1.8155031204223633, -0.0748421773314476, -0.12904593348503113, -0.011966477148234844, 1.1934534311294556, 0.6014008522033691, 0.5702534914016724, 0.3342611789703369, 1.0812885761260986, -0.6028545498847961, 0.41166892647743225, 0.19711479544639587, -0.7572340369224548, 0.8509772419929504, -0.939834475517273, 0.597562849521637, 0.08730323612689972, 1.9627240896224976, -0.2547740638256073, 0.10142695903778076, -0.39880961179733276, -1.1482431888580322, -0.14908446371555328, 0.6586464643478394, 0.7799766659736633, -2.612487554550171, 0.5127801895141602, 0.4238651990890503, -0.7212640643119812, -0.9599864482879639, -0.3601917624473572, 0.7787762880325317, -0.060107216238975525, -0.989209771156311, 0.3416585624217987, -0.5660362839698792, 0.794161856174469, -0.90653395652771, 0.9134836196899414, 1.0714350938796997, 0.26946863532066345, 1.6046206951141357, 0.5918529033660889, -0.38803714513778687, 0.39793193340301514, -0.8600447177886963, 0.7611907720565796, -2.8103299140930176, -0.1351856291294098], [-0.1491370052099228, -1.4619704484939575, 1.8823212385177612, 2.249241352081299, 0.35704654455184937, 0.44543030858039856, -0.624301552772522, -1.7510313987731934, 0.9111815690994263, -0.7833273410797119, 0.3691873252391815, -0.4031387269496918, 1.2888814210891724, 1.1618752479553223, 0.12417841702699661, -0.2893272936344147, -0.15679287910461426, 0.5353083610534668, 0.8020332455635071, -2.4236738681793213, 0.7241044044494629, -0.33752578496932983, 0.8078498840332031, -1.356379508972168, -0.9197118878364563, -1.303173303604126, 1.579001784324646, -0.08220133185386658, -0.6128100156784058, 0.27134981751441956, 1.5617729425430298, 1.1707876920700073, -1.0934444665908813, 0.9559402465820312, 0.3857380747795105, 0.5673568844795227, -0.549008309841156, -0.6086229085922241, 1.808953046798706, -0.24009908735752106, 0.30441567301750183, 0.30260753631591797, 0.49649617075920105, -0.649394154548645, 0.7308369278907776, -0.09548389911651611, 0.640999436378479, -0.6705748438835144, 0.16815373301506042, -0.20981620252132416, -0.5429834723472595, 1.4508317708969116, 0.47712093591690063, 0.1499660611152649, 0.07325150817632675, -0.31164050102233887, -0.0880456417798996, 1.8128811120986938, 0.4910730719566345, 1.014730453491211, -0.0487215593457222, -0.048955850303173065, -2.0136313438415527, -1.1425509452819824, 0.7933687567710876, -0.22523649036884308, -0.4384666085243225, 0.4625057280063629, -1.3913735151290894, -1.0618906021118164, 1.645633339881897, 0.10051698982715607, -0.8240103721618652, 0.7249805927276611, -1.2477316856384277, -0.5762883424758911, -1.905778169631958, -1.149836540222168, 0.30712127685546875, 0.0774398222565651, -0.449325829744339, -0.5606898665428162, -0.020443519577383995, 0.7154061198234558, -0.6378291249275208, 1.1316452026367188, -1.2289433479309082, -0.7246579527854919, 0.29436251521110535, -0.7955834865570068, -0.2828541100025177, 1.1685030460357666, 0.9012662172317505, -0.042802032083272934, 0.4398089051246643, 1.802321195602417, 0.16653376817703247, 0.3092016875743866, -0.029546303674578667, -1.1660282611846924, -1.0024200677871704, 0.01498194970190525, -0.05033785104751587, 1.288622498512268, 0.7936789989471436, 1.6705235242843628, 0.11338990181684494, 0.29714062809944153, -0.5986618995666504, -1.2596709728240967, 0.9044061899185181, -0.7769901156425476, 2.062798500061035, -0.2069290578365326, 1.0808221101760864, -0.8567869663238525, 0.8263351321220398, -0.12076963484287262, -1.5471190214157104, -0.03036302514374256, -0.14701038599014282, -0.8011499047279358, -1.5100938081741333, -0.003508168738335371, -1.4193847179412842, -0.06649920344352722, 0.5462164878845215, -0.25979065895080566], [-0.724210262298584, -0.6249135732650757, -1.6962859630584717, -1.511947751045227, 0.41366320848464966, 0.807019054889679, -1.5825577974319458, -2.3853933811187744, 1.4075005054473877, 0.3788139224052429, -0.0050802831538021564, 0.304584801197052, 1.2031222581863403, 0.15064960718154907, 1.0599133968353271, -0.2649408280849457, 0.924005389213562, -0.7822390794754028, 0.248155876994133, -0.1825110763311386, 1.3516377210617065, -0.5806597471237183, 1.1039788722991943, -1.4335589408874512, 0.1996026635169983, -0.11556387692689896, 0.7082404494285583, -0.3049173951148987, 0.6646769642829895, 1.7287336587905884, -1.0164072513580322, -0.8800082206726074, 1.294896125793457, 1.2052960395812988, 1.2311663627624512, 0.012019955553114414, 0.8399214148521423, 0.3000200688838959, -1.6797478199005127, 0.10458926856517792, 0.9180344343185425, -0.837556004524231, -0.5856409668922424, -2.8114607334136963, 0.20135818421840668, -0.010018838569521904, 0.34196871519088745, -0.4002983570098877, 0.25664183497428894, -0.28305354714393616, -1.5448212623596191, 1.179274320602417, -0.08057665824890137, 1.3541682958602905, -1.2571583986282349, -1.9406989812850952, -1.446834683418274, 2.909457206726074, -0.8221254944801331, 1.761320948600769, -1.8535850048065186, 0.2992779314517975, -1.4114320278167725, -0.19246940314769745, 0.1837523728609085, 0.010196120478212833, -0.08926122635602951, 1.1929570436477661, -1.3665306568145752, 0.3301510810852051, -0.24452686309814453, -0.42529186606407166, 0.6241697072982788, -1.3649142980575562, 0.46339255571365356, 0.05446617305278778, -0.9790117740631104, -0.5114567279815674, 0.6676783561706543, 0.30410975217819214, 0.592513918876648, -0.3895128667354584, 0.4750294089317322, -2.0375192165374756, 0.004464444704353809, 2.8509521484375, 1.7740983963012695, 1.8109397888183594, -1.2647498846054077, -0.7116406559944153, 0.49971431493759155, -0.019686121493577957, 1.5649935007095337, 1.655333161354065, -0.8313153386116028, 0.28453636169433594, -0.2925979793071747, -0.39784446358680725, 0.06810946017503738, -2.1613101959228516, 0.7067722082138062, 1.9119611978530884, 0.824833869934082, -0.1990535706281662, 1.5069431066513062, 0.7909429669380188, -2.814054012298584, 0.5027386546134949, -0.415810227394104, 0.8288232684135437, 0.91844642162323, 0.43168413639068604, 1.5933079719543457, 0.3297702372074127, -0.3548380732536316, 1.7602232694625854, -0.19952064752578735, 1.7311843633651733, -1.1820961236953735, 0.356334388256073, -1.1774322986602783, -0.2924216687679291, -0.6459290385246277, 2.637514114379883, -0.591498076915741, 0.7260568141937256, 0.07603573799133301, -0.15025879442691803]], [[0.09411745518445969, -1.8039681911468506, -0.26056385040283203, 0.5139756202697754, -0.968822181224823, 0.9975171685218811, 1.062661051750183, -0.6219176650047302, 0.33292070031166077, -0.29755595326423645, -0.5528262853622437, -0.5316406488418579, -0.25182846188545227, -2.3427388668060303, 0.1171228289604187, 1.5124592781066895, 1.2247906923294067, -0.5701493620872498, -0.9747312664985657, -0.26204782724380493, 0.5920752882957458, 0.6082141995429993, -2.2086400985717773, -0.16437794268131256, -0.499415785074234, -0.1457509696483612, 1.5887752771377563, -2.037017345428467, 0.05588950961828232, -0.7578247785568237, 0.6700415015220642, 0.4137488603591919, -0.6719561815261841, 1.4908246994018555, 0.2334296703338623, -0.1968475729227066, 1.533532738685608, -0.44914719462394714, -1.6018940210342407, -2.280806303024292, -0.09961346536874771, 0.17517894506454468, 1.1477018594741821, -0.5966320633888245, -0.7070205211639404, 0.28372862935066223, -1.1100081205368042, -0.05845797434449196, -0.884250819683075, -0.18731705844402313, 0.31423741579055786, 0.40202170610427856, -0.2702200412750244, 0.5941079258918762, 0.4328450560569763, 0.9324436783790588, 0.335870623588562, 0.14230260252952576, 0.7676709890365601, -2.313652276992798, -0.12513454258441925, 1.3788251876831055, -0.04731062799692154, 1.8950849771499634, -0.5041353702545166, -1.1604325771331787, -1.6986274719238281, 1.31460702419281, -0.5628029704093933, 0.18562696874141693, -0.19237075746059418, 1.0930038690567017, -0.11540328711271286, -0.22852784395217896, -1.6732475757598877, 0.0008627046481706202, 0.6366298794746399, -0.6563210487365723, 0.12099812179803848, -0.6856749653816223, 0.48954567313194275, -0.4085141122341156, 0.5777456760406494, -0.1418226957321167, 1.168122410774231, -1.09257972240448, 0.342278391122818, -0.6299504637718201, -0.16708533465862274, 0.7691465616226196, 0.07124242931604385, 1.601225733757019, -0.9407801628112793, 0.482477068901062, -0.44908270239830017, 0.11823786050081253, 1.3146748542785645, -0.6956305503845215, 1.5694725513458252, -0.7574234008789062, -0.07923140376806259, 0.4142260253429413, -0.768262505531311, -0.8977240920066833, -0.4325005114078522, 2.5634050369262695, -1.3218661546707153, 1.587430715560913, -0.4890959560871124, -1.1305851936340332, -0.3408682644367218, -1.2043156623840332, -2.5129568576812744, 0.06429243087768555, 0.5416719913482666, 1.2239423990249634, 1.1164827346801758, -0.507577121257782, -2.1596720218658447, -0.6762502193450928, 1.440150260925293, -1.753095269203186, -1.1185765266418457, -0.5946805477142334, -1.6255829334259033, -0.7570950984954834, 1.1144850254058838, -0.04026593267917633], [0.2447660118341446, 0.46149685978889465, -0.6329536437988281, 0.016282781958580017, -0.5741618871688843, 1.663793683052063, 0.3732549548149109, 1.2900111675262451, 0.4414823651313782, 0.3355304002761841, 1.0846011638641357, 0.39118561148643494, 1.4265680313110352, -0.7462683916091919, -1.255879521369934, 0.6091519594192505, 1.1265870332717896, -1.2613248825073242, 0.3343755304813385, 0.67598956823349, -0.22581076622009277, -0.9352304339408875, 0.3299289643764496, -0.652159571647644, 1.213390588760376, -0.9772459864616394, 1.1326323747634888, 1.9175758361816406, -1.0538325309753418, -0.48536792397499084, 0.546653687953949, 0.32358309626579285, -0.5168946981430054, 0.1429300308227539, -0.6784319281578064, -1.264905571937561, -0.3180784285068512, 0.4587738513946533, -0.474417120218277, -1.7906079292297363, 1.2643296718597412, -0.7729059457778931, -1.2293612957000732, 1.9819409847259521, 1.7368422746658325, 0.21039573848247528, 0.42564859986305237, -0.07123042643070221, -0.14737656712532043, 0.17470835149288177, 0.24471884965896606, 0.7736198902130127, 1.0977498292922974, -0.31954342126846313, -0.8174154162406921, 0.254851371049881, -0.15014128386974335, 1.4376846551895142, 0.625285267829895, -0.7344803810119629, -1.4023194313049316, -1.7066563367843628, -0.9034457802772522, -0.9018018245697021, 0.8881639838218689, 1.2950564622879028, 0.24056456983089447, -1.1363346576690674, 0.18500953912734985, -0.30045104026794434, 1.8223111629486084, -1.4997953176498413, 0.6149839758872986, 0.30297645926475525, -1.1741245985031128, 1.1478029489517212, -0.9413424730300903, 0.4232901334762573, -1.6124229431152344, 0.84578537940979, -1.4744799137115479, -0.0038975835777819157, -0.7647051215171814, -0.03765695542097092, -0.5710165500640869, 1.5024480819702148, -1.5165537595748901, 0.8699204325675964, 0.5272228121757507, -0.7842587232589722, -0.5282149910926819, 2.847433090209961, -0.01349780336022377, 1.568420171737671, -0.45486775040626526, -0.8564525246620178, 0.25411728024482727, -0.18911541998386383, -0.390651673078537, -0.23872818052768707, 0.9469940662384033, 0.6938055753707886, -0.6455655694007874, -0.43607479333877563, 0.2769109308719635, -0.07261822372674942, 0.26332712173461914, -0.46117573976516724, -0.3704869747161865, 1.4907417297363281, 0.38474875688552856, -0.14857080578804016, -0.9429118037223816, 0.38150498270988464, 0.07289273291826248, -1.7642751932144165, -0.5847663879394531, -0.11590786278247833, -1.5001065731048584, 0.5212512016296387, 0.13506673276424408, 1.3968472480773926, 1.176142692565918, -0.526031494140625, -0.06274305284023285, 0.11328642815351486, 0.17628838121891022, 0.7285598516464233], [-0.005983575712889433, 1.4113878011703491, 1.3026248216629028, -0.500929594039917, -1.561394214630127, 0.5884178280830383, -0.322958379983902, 1.3621374368667603, -0.16986852884292603, 0.9105108976364136, -0.997446596622467, 0.7354539036750793, -0.9282095432281494, -1.6127299070358276, 0.4022921025753021, -0.8604150414466858, -0.26902246475219727, -1.5185744762420654, 1.143295168876648, -0.7163010835647583, 1.5993711948394775, 1.3431707620620728, -1.4239795207977295, 0.6959758996963501, 0.5253583788871765, -0.981579601764679, 1.056796669960022, 0.2810480296611786, -1.0208477973937988, -0.25879403948783875, -0.4138047695159912, -0.8727529644966125, 0.649116039276123, -0.6042114496231079, 0.2977234125137329, 0.3929271399974823, 0.3013208508491516, -3.268073081970215, -0.14100857079029083, 0.42125654220581055, -1.1712528467178345, -0.9012458920478821, 1.089680552482605, -1.606480360031128, 0.9138601422309875, -1.1329759359359741, 1.650467038154602, -0.5823144316673279, -0.47989267110824585, 0.3450123965740204, 1.063159704208374, -0.7307308316230774, -1.2490262985229492, -0.8845866918563843, -0.04186048358678818, 1.197682499885559, 2.225651264190674, 0.16056238114833832, 0.0899413526058197, -0.12310449779033661, 0.8642337918281555, -0.4010537266731262, -0.2296198457479477, 2.0688400268554688, -0.6328169703483582, -0.6265005469322205, 0.3348322808742523, 0.6128578782081604, 1.1238338947296143, -0.9096003174781799, 0.9335702061653137, 0.6663300395011902, 0.9757553339004517, 0.912036120891571, -2.6298515796661377, 1.0548526048660278, -0.8280993103981018, -1.079919457435608, 0.3251344859600067, 1.451873540878296, -0.8332589268684387, 0.28009700775146484, -0.34200745820999146, 2.015108585357666, 0.7498697638511658, 0.6835613250732422, 1.6265771389007568, 0.17737580835819244, -2.122440814971924, -0.4784508943557739, -0.19570429623126984, 0.23951533436775208, 0.23938299715518951, -1.0677915811538696, -0.6498832106590271, 0.5250700116157532, -0.454169362783432, 0.4318963587284088, 0.17633487284183502, -0.2935008108615875, -1.4708611965179443, 0.3695472478866577, 0.07208939641714096, -1.7056591510772705, 2.092874765396118, 0.07058345526456833, 0.09973906725645065, -0.168464794754982, -0.2742786705493927, -0.834926187992096, -2.200976610183716, -1.6718018054962158, 0.11635429412126541, -0.6007007360458374, -1.4233779907226562, 0.13440275192260742, -1.6381677389144897, 0.04978662356734276, 0.471962034702301, 1.2774611711502075, -0.9335002899169922, -0.4142555594444275, -1.5046412944793701, -0.39824041724205017, -0.03614027053117752, -1.5325006246566772, 1.6596604585647583, 0.21684758365154266], [-0.746888279914856, 0.7298562526702881, 0.4409979581832886, 0.32240575551986694, -0.5068615674972534, -0.1099507063627243, -0.07523749023675919, -0.2412519007921219, -0.6426459550857544, -0.2876071631908417, -0.031741175800561905, -0.6736345291137695, 0.6395487189292908, -1.363205075263977, 0.8768255710601807, 0.3300009071826935, -1.3371435403823853, -0.6010357737541199, -0.735184371471405, -1.1066561937332153, 1.1969654560089111, -1.0484325885772705, 1.279737114906311, 1.3861967325210571, -0.712529182434082, 0.788497269153595, 2.6479434967041016, 0.2884577810764313, 0.30054277181625366, 0.040359679609537125, 1.0573474168777466, -1.2523562908172607, -0.8901972770690918, -0.8643385171890259, 0.5318257808685303, -2.421281337738037, -1.5041459798812866, 1.8722584247589111, 1.6525204181671143, -0.6830832958221436, -1.1568368673324585, -0.0903732180595398, 1.1590524911880493, -0.2994539439678192, 0.28556567430496216, -0.21245603263378143, -1.0998256206512451, -0.5001055002212524, -0.3855637311935425, -0.062289975583553314, 0.2073240876197815, 2.0462653636932373, 0.37170684337615967, 1.6573338508605957, 0.17888200283050537, -0.26844385266304016, 0.9778140187263489, -0.07905369997024536, 1.5689847469329834, 0.9620531797409058, -0.5693052411079407, 0.5346065163612366, -0.18252870440483093, -0.08803923428058624, -0.9531571865081787, 0.9434134364128113, -0.49843212962150574, 1.5422935485839844, 1.2469931840896606, 0.61323082447052, 0.9304488897323608, -0.663870632648468, -1.5156468152999878, 1.1749107837677002, 1.8997429609298706, -0.941077470779419, -1.451339840888977, -0.33299046754837036, 0.2240966558456421, -0.30898579955101013, -0.7946396470069885, 0.4903898537158966, 0.05021626502275467, 0.5812122821807861, -0.7856221795082092, 0.7481939196586609, -1.156384825706482, 0.13137930631637573, -1.6163418292999268, -0.4969625473022461, -1.799583077430725, -0.19141459465026855, -0.7438542246818542, 1.076704740524292, -0.00678902817890048, 0.9076809287071228, -0.25024059414863586, 0.15363112092018127, -0.08317896723747253, 0.8304197788238525, 0.7692674994468689, 0.05631069839000702, -0.3233530521392822, 2.1239988803863525, -0.8184458017349243, -1.7135189771652222, -0.33635956048965454, 0.5608400702476501, 0.5498939156532288, 0.48662084341049194, -0.8377702832221985, 1.2306474447250366, -0.4146026074886322, 0.3527143597602844, -1.660675287246704, 1.7469191551208496, -1.0882740020751953, -0.8249647617340088, 0.4138681888580322, -0.6666907668113708, -0.8569666147232056, 1.1359670162200928, 0.7708715796470642, -0.4923759996891022, -1.4059081077575684, -0.2521410286426544, 0.9531877040863037, -0.998929500579834], [-0.8127986788749695, -0.36148738861083984, 0.5371900200843811, 0.20098018646240234, -0.33263683319091797, 0.0964946448802948, -1.574344277381897, 0.6015411019325256, -0.7129696607589722, 0.8790431618690491, -1.0876582860946655, 0.7351844906806946, 1.0068422555923462, -2.046780824661255, 0.7031422257423401, 0.1013716608285904, -0.488333135843277, -1.8387757539749146, -0.23421594500541687, 0.5118805170059204, 1.8211220502853394, 1.3325440883636475, -0.41038504242897034, 0.5135353207588196, 0.09970255196094513, -0.5627663135528564, 1.1247978210449219, 0.7332815527915955, -0.2471417784690857, -1.7701244354248047, -0.4719083905220032, -2.197760581970215, 0.6346349716186523, -0.1056683138012886, 0.6745113730430603, -0.5943995714187622, -0.48358821868896484, 0.7805432677268982, 0.48708873987197876, 0.9383949637413025, 1.5160436630249023, -0.23742477595806122, 3.0349199771881104, 1.8527414798736572, 1.1190943717956543, 0.26148924231529236, 0.15658308565616608, -0.3155013918876648, 0.8725534677505493, 0.18668535351753235, -0.7716221213340759, 0.41277390718460083, -1.190142273902893, -0.08158349245786667, 0.41610240936279297, -1.3878344297409058, 0.08636525273323059, -0.8587873578071594, -0.7802782654762268, 0.9993433952331543, -0.261572927236557, 0.37461918592453003, 0.6475968360900879, -0.3107379078865051, 1.0295268297195435, -0.33435797691345215, -2.1279702186584473, 0.31876441836357117, 1.8369981050491333, -0.2150164395570755, 0.8350484371185303, -0.41702985763549805, -0.41518592834472656, -1.0776054859161377, 0.7671626806259155, -1.8446074724197388, -0.4545789957046509, 0.9870574474334717, -0.2844054400920868, -0.08870209008455276, -0.11339333653450012, 0.7613571882247925, -1.6924160718917847, -0.2070159614086151, -0.2328338921070099, 0.610177218914032, 0.3338358998298645, -2.24845027923584, 0.7967739701271057, -0.14951829612255096, 1.3789350986480713, -0.33148667216300964, -0.1798057109117508, -0.6482121348381042, 0.9525177478790283, 0.407888799905777, -0.629298210144043, 0.7073100209236145, 0.2831636071205139, -1.4297393560409546, -1.5126820802688599, -0.6686965227127075, 0.8393918871879578, 0.5706257820129395, -0.5382395386695862, 0.5119798183441162, -0.3612027168273926, 0.062350574880838394, 1.066825270652771, 0.5549873113632202, 0.6618029475212097, 0.4625619649887085, -0.43457114696502686, -0.08568233996629715, 0.4412204623222351, -1.3806196451187134, -0.49950772523880005, -1.0075136423110962, 0.6664385795593262, -1.4418412446975708, 0.41759952902793884, 0.7712258100509644, 0.5829719305038452, -0.4046511948108673, -0.01638942025601864, -1.2932802438735962, 1.6605098247528076, -0.19664549827575684], [0.6421281695365906, -0.021849602460861206, -0.001022180775180459, -0.4333014190196991, -0.9217462539672852, -1.214048147201538, 0.030446801334619522, -0.18715429306030273, -0.43001192808151245, -0.513929545879364, -0.9442569613456726, -1.3738980293273926, -1.0484586954116821, -0.20869742333889008, -1.5167872905731201, 0.41430461406707764, 1.4012378454208374, 0.6871628761291504, 0.6603675484657288, -1.1983414888381958, 0.9192841053009033, 0.13932472467422485, -1.4331196546554565, 0.5926997661590576, 0.2564004361629486, 1.3184102773666382, -0.006875971332192421, 1.654868483543396, -0.2527613341808319, 0.18441496789455414, -0.26495927572250366, 0.022539611905813217, 0.6850605010986328, -0.7985510230064392, -0.8201749324798584, 1.1353679895401, 1.5526472330093384, -0.4526137709617615, -0.324421763420105, 0.24332402646541595, 0.7535991072654724, -0.23630085587501526, 0.5868064761161804, -0.9501832723617554, -0.7913445830345154, 0.37920013070106506, -0.01373518817126751, 1.6362850666046143, -0.5761903524398804, -0.5402950644493103, 0.8465831279754639, 1.3568463325500488, 0.36223167181015015, -0.7272855043411255, -0.7099135518074036, 0.04894379526376724, 0.6045421957969666, -1.5932427644729614, -0.6223369836807251, -0.5815420150756836, 0.8106145858764648, 0.6799525022506714, -1.5674015283584595, 1.5193157196044922, 1.0184640884399414, 0.5882672071456909, -0.8392923474311829, -0.40046027302742004, 0.3514900803565979, -1.0378670692443848, -1.0605653524398804, -0.31615784764289856, 0.2249295711517334, 0.4953032433986664, -0.5108792781829834, -1.0911864042282104, 1.195855975151062, 1.003664493560791, 0.8794687986373901, 1.0618277788162231, -0.4257408082485199, -0.6296576857566833, 1.45114266872406, 0.6911429762840271, 0.940601646900177, 0.0677395835518837, 1.2658536434173584, -0.31687095761299133, 0.8571318984031677, 0.753031313419342, 0.24817128479480743, 1.4058197736740112, 0.18959760665893555, -1.3837742805480957, 0.6726384162902832, -2.5304338932037354, -0.7487031817436218, 0.01881173439323902, 0.4782506227493286, -1.0793378353118896, -0.9447135925292969, 0.614636242389679, 1.434095859527588, 1.71779203414917, -1.4816538095474243, -0.2608443796634674, -1.10720956325531, 0.7119618058204651, 0.18921014666557312, -0.21341557800769806, 0.9638840556144714, 0.9396551847457886, 0.5051775574684143, 0.719861626625061, -0.41441231966018677, -1.064624547958374, 0.09127480536699295, 0.8539327383041382, 0.8231799006462097, -0.3009321391582489, -0.4255560636520386, 1.10575532913208, -0.3355211019515991, 0.17024153470993042, 0.12316272407770157, 0.011648214422166348, 1.1286591291427612, -0.38424578309059143], [-0.15475256741046906, -2.429672956466675, -1.7769451141357422, 1.936871886253357, -0.1061733067035675, 0.4379099905490875, 0.2378869503736496, 0.26302796602249146, -0.03720900043845177, -1.4198591709136963, 0.8120472431182861, 0.7924329042434692, -0.8403787612915039, 1.5996507406234741, 0.2522849142551422, 2.756148338317871, -0.9618062376976013, 0.21897418797016144, 1.0501627922058105, 0.7955169677734375, -0.2112501561641693, 1.104688048362732, 0.6456941366195679, 1.1682482957839966, -2.1231348514556885, 1.3566219806671143, -0.19394440948963165, -1.4119995832443237, -0.5038023591041565, -0.5830456018447876, 0.936646580696106, -0.817937433719635, -0.30624911189079285, -1.5835720300674438, -0.19278401136398315, 0.6747621297836304, -1.5823042392730713, -0.7062907218933105, 0.6429787278175354, -0.020199796184897423, 0.8140385150909424, -0.0298044141381979, 0.0949544906616211, 2.0466227531433105, 0.8747974634170532, 0.46922415494918823, -0.5785823464393616, 0.2936009168624878, 0.7892258167266846, 0.3999788463115692, 0.4563497304916382, -0.7161324620246887, -0.4946957528591156, 0.2986937463283539, 0.627013623714447, -0.629084050655365, -0.9915807247161865, -1.2860286235809326, -0.24806509912014008, 0.36266663670539856, -0.7539509534835815, -0.9159796833992004, 0.5129892826080322, -0.7659558653831482, 0.5141393542289734, -0.11397112160921097, 1.6020647287368774, 0.020716119557619095, -0.90525883436203, 0.22997990250587463, -0.6584793329238892, -1.1671454906463623, -0.4471031129360199, -1.6639723777770996, -0.30215442180633545, -0.5647557973861694, -0.9410176277160645, 1.0226576328277588, -0.6106995344161987, 0.3649923503398895, 0.9947285056114197, 0.03860161826014519, -1.727194905281067, 0.11710890382528305, 0.5540628433227539, -0.6221849918365479, -0.07920794934034348, 0.27891483902931213, -0.6497475504875183, 0.4105985462665558, -1.3172461986541748, -0.5925548076629639, -0.5704255700111389, -1.3550384044647217, -0.653475821018219, 1.5846283435821533, -1.4638584852218628, 0.7433646321296692, 1.8847694396972656, 0.5973578691482544, -0.6509997844696045, 0.9259183406829834, -1.3315685987472534, -0.36907973885536194, -1.9681689739227295, 1.2843749523162842, 1.6721657514572144, -0.4277040362358093, -1.880458950996399, -0.7021753787994385, -0.610638439655304, -0.4559808671474457, -0.9712055325508118, 0.4270852208137512, 0.22426185011863708, 0.831221342086792, -0.8870702981948853, -2.3134922981262207, -0.27020373940467834, -0.6488902568817139, -0.9612461924552917, 2.6707146167755127, -0.05434567853808403, -0.7657480239868164, -0.6781557202339172, -1.6716364622116089, 0.2749159038066864, -0.7247098684310913], [1.2423346042633057, -1.2162114381790161, -0.5066167116165161, -2.464714288711548, 0.26358848810195923, -0.37629279494285583, -1.4421685934066772, 0.921477735042572, 0.09334421157836914, -0.27813562750816345, 0.6093566417694092, 0.25597405433654785, -1.1347620487213135, 0.08171357214450836, -1.6303685903549194, -1.8836194276809692, -0.5594058036804199, -1.6281795501708984, -0.23441952466964722, -0.316561758518219, 0.14205671846866608, 0.47753068804740906, -0.7235743403434753, 1.4421846866607666, 2.332859516143799, 2.0110292434692383, 0.7827330231666565, 0.02677486650645733, -1.0192714929580688, -0.07165390998125076, 1.9965486526489258, -0.3086024820804596, -1.66944420337677, -1.8582026958465576, -1.8160985708236694, -0.9187415242195129, -0.5842670202255249, 1.7428979873657227, -0.0742974504828453, -1.2050846815109253, 0.2943462133407593, 1.9398624897003174, -0.8294176459312439, -0.1597256064414978, 0.22721998393535614, 1.9069626331329346, 1.3266505002975464, 0.9086805582046509, 1.7713212966918945, 0.865088164806366, 0.7455442547798157, 1.4102580547332764, -2.177131414413452, -0.7302797436714172, 0.07073379307985306, -0.8011468648910522, -0.4778078496456146, -0.48804011940956116, 2.2528867721557617, -0.11808308213949203, -0.5701113343238831, -1.027744174003601, 0.6918261051177979, -0.6323533654212952, -0.08256645500659943, 1.345570683479309, 0.09425579011440277, 0.7311151027679443, -1.6896281242370605, 0.8267673254013062, -0.49218714237213135, -0.1965983361005783, -0.9964653253555298, 1.2324182987213135, -0.32579100131988525, 1.002626895904541, -0.21349121630191803, -1.0941303968429565, 1.0979583263397217, -1.2200998067855835, 0.42501378059387207, 0.16615413129329681, -0.5841394066810608, 1.7199122905731201, 1.1082721948623657, -1.3659480810165405, -0.29406020045280457, -2.1965088844299316, 0.0383475199341774, -2.3257129192352295, 1.0469281673431396, 0.7162013053894043, -1.471768856048584, -0.47130003571510315, 1.0149524211883545, -0.42011839151382446, -0.4546459913253784, -1.2286995649337769, 1.3304736614227295, -1.2942044734954834, -0.6487411856651306, -0.04542265832424164, 0.12525729835033417, -0.18465964496135712, -0.8062379956245422, -1.1008679866790771, 0.8979679346084595, -0.48982706665992737, -1.425523042678833, -1.1439762115478516, -0.8468969464302063, 0.6386964917182922, 2.1830904483795166, -0.6758313775062561, -0.8249437212944031, -0.3680219054222107, 0.20905378460884094, 0.8978037238121033, -0.5311604738235474, -0.7511839270591736, 0.4149515628814697, -1.5329034328460693, -0.2753383219242096, 2.1282167434692383, -1.2116986513137817, 0.26076918840408325, 0.1857263147830963, -0.5645031332969666]], [[1.037593126296997, 1.359885573387146, -1.0420479774475098, -0.766249418258667, -1.179237723350525, -0.34704115986824036, -0.7496975660324097, -1.6205554008483887, 0.5365379452705383, -0.9022252559661865, 1.5292514562606812, 0.5766884684562683, -0.13513007760047913, 0.7111337780952454, 2.3242762088775635, -0.7970829606056213, -1.3063427209854126, -0.11705004423856735, 0.2558038532733917, 0.8438387513160706, 0.6107924580574036, 0.3462335467338562, -0.5173033475875854, 1.3917566537857056, 0.9448468089103699, -0.2194366753101349, 0.38720861077308655, -0.8194106221199036, -0.1527930051088333, 0.9424098134040833, 1.4902033805847168, 0.7096362113952637, -1.1006709337234497, -0.42587634921073914, 0.6086241006851196, -1.5334855318069458, -2.0921733379364014, 0.43259453773498535, 1.0822110176086426, 2.7113664150238037, 1.5087459087371826, 0.36600878834724426, -1.9312493801116943, -1.6355575323104858, 1.318080186843872, -0.7098495960235596, -1.1264803409576416, 0.12634196877479553, -1.1373229026794434, 0.25217878818511963, -0.8516297936439514, -0.25038617849349976, -2.228759765625, -0.556468665599823, -0.4294205605983734, -1.3413772583007812, 0.9324784278869629, -1.911824107170105, 0.9500294923782349, 0.6509804725646973, 0.11776456981897354, -0.693302571773529, -0.39239004254341125, 1.6800079345703125, -1.0540772676467896, -0.696898341178894, 0.3953584134578705, -0.9087831377983093, -0.6093488931655884, 0.7678235173225403, 0.658776581287384, -0.9036016464233398, 1.5860131978988647, -0.33554771542549133, 0.4300939440727234, -0.7487747073173523, 1.3359150886535645, 2.3140361309051514, 2.1377828121185303, 0.003169435076415539, 0.026961615309119225, 0.6993094086647034, -1.1436365842819214, -1.9987964630126953, 1.6225353479385376, -0.07280930131673813, -0.5333480834960938, 1.1948482990264893, -0.331461101770401, 0.8047428727149963, 0.776978611946106, -0.6483163237571716, 1.502239465713501, 1.1350098848342896, -0.5361555218696594, -0.3638506233692169, 1.6284680366516113, 0.5358354449272156, 1.27030611038208, 0.433001309633255, -0.8031932711601257, 0.44511663913726807, -0.38059207797050476, -0.13730350136756897, -0.5564236640930176, 0.39557158946990967, -0.0010935653699561954, -1.336909532546997, 1.2294011116027832, 2.2162294387817383, 0.47931012511253357, 0.7551690340042114, -0.3934750556945801, 1.1222485303878784, -0.47616100311279297, -1.1142616271972656, 1.3326467275619507, 0.748676061630249, -1.3312702178955078, -1.4788497686386108, 0.23508530855178833, 0.24087005853652954, -2.030667543411255, 0.8801771998405457, 0.8543329834938049, 0.08398047834634781, 0.8750160336494446, 1.6555202007293701], [-0.24521033465862274, -0.4598110616207123, -2.327113389968872, 1.4809727668762207, -0.8794218301773071, 1.0599967241287231, -1.4167073965072632, 2.3351852893829346, 1.0073710680007935, 0.26825714111328125, 0.9808425903320312, -1.7323836088180542, 2.6578726768493652, -0.5221295356750488, -0.39704468846321106, 1.1404554843902588, 0.8685501217842102, 0.4941859543323517, -1.1606634855270386, -1.7508543729782104, 0.29367581009864807, -0.48432841897010803, -0.8121354579925537, 0.8564830422401428, -0.8117498755455017, 0.3495715260505676, 1.9690736532211304, -0.18587395548820496, -0.35201555490493774, 0.9690021276473999, -0.04805208742618561, 0.5133786797523499, 0.7922642230987549, 1.4660601615905762, 0.6384323835372925, -0.9698423743247986, 0.9350240230560303, 1.8108397722244263, 1.1622087955474854, 0.3712543845176697, -0.6814975142478943, 1.67984938621521, -0.5387322306632996, 0.12224645912647247, -0.43324586749076843, 0.6027276515960693, 0.2309790551662445, 0.21149548888206482, -1.327663779258728, -0.8690558671951294, -0.5311740636825562, 0.8329710960388184, 0.8704860210418701, 0.4168357849121094, 0.49360883235931396, 0.060155075043439865, -0.6761301755905151, -0.015836821869015694, -0.9061166048049927, 0.017732134088873863, -0.25036105513572693, 0.25000086426734924, -0.07879388332366943, 0.778153657913208, -1.116320013999939, 0.5844866037368774, -0.3142145276069641, -0.968712329864502, 1.3995801210403442, -1.3984990119934082, 0.044220853596925735, 1.6627088785171509, -1.8813457489013672, -0.7706892490386963, -0.09237031638622284, -1.2415226697921753, 1.5012613534927368, 1.956005573272705, -1.393769383430481, -1.3939757347106934, 0.40506142377853394, -0.8503071069717407, 0.9452424645423889, -0.15660181641578674, 1.5255982875823975, 1.8235706090927124, 0.39946866035461426, -0.7877188920974731, 0.0011566767934709787, -2.303999185562134, 0.660001277923584, -1.5092594623565674, -0.1828344762325287, 1.6267428398132324, -1.3792892694473267, 0.37346744537353516, -1.3435035943984985, -0.8653835654258728, 0.16578561067581177, 1.161827802658081, -2.5105741024017334, -1.198278784751892, -0.0993141457438469, -0.761955976486206, -0.3813022971153259, 0.4482186734676361, 1.3697813749313354, -0.020679855719208717, 1.0688668489456177, -0.6234690546989441, 1.0425828695297241, -1.092887043952942, -0.32872316241264343, -0.6699380278587341, 1.253564476966858, -1.8148589134216309, -1.8228631019592285, 0.5765841603279114, -0.4539690315723419, 0.44402721524238586, 2.1862330436706543, 0.8103130459785461, -0.5830438733100891, 0.43714287877082825, -0.8104554414749146, -1.3415427207946777, -1.21769118309021, -1.0583670139312744], [-0.18821264803409576, -2.4321963787078857, -0.30376335978507996, 0.3671421408653259, 1.244457483291626, -0.08187976479530334, -0.04228805750608444, -1.464524507522583, 2.297720193862915, 0.4989206790924072, 1.1758209466934204, -0.8869984745979309, 0.7800905704498291, -0.03415989130735397, -0.7738083004951477, -0.8740308880805969, -0.6744934320449829, -0.8093104958534241, 0.7306948900222778, -0.17997321486473083, 2.0948596000671387, 0.3110545575618744, 1.0605417490005493, -0.1752137541770935, -1.1145187616348267, 0.041338589042425156, -0.15569862723350525, -0.865635097026825, 1.3614821434020996, -0.6321187019348145, -0.8358040452003479, -0.35383617877960205, 0.14496253430843353, 1.1800918579101562, 1.5604918003082275, 0.1041029840707779, -0.7111517786979675, -0.1646192967891693, 0.639728844165802, -1.1514493227005005, 2.79848313331604, -0.6732430458068848, 0.8102198839187622, -1.075445532798767, -2.013913631439209, 0.07816876471042633, 1.0671063661575317, 0.5544818043708801, 0.15462106466293335, -0.48469799757003784, -1.4833165407180786, 0.040034063160419464, -0.2897835969924927, -0.4626242518424988, 0.3276630640029907, -0.10294710099697113, -0.05492227151989937, 1.6736857891082764, 1.5282212495803833, -1.4152626991271973, -1.2233774662017822, 1.1973662376403809, -1.8518563508987427, 2.251065969467163, 0.2783076763153076, 1.4395294189453125, 1.6026651859283447, -1.0518689155578613, 1.5192731618881226, -0.0910818800330162, -1.2114650011062622, -0.5258087515830994, -1.739113688468933, 0.9642657041549683, -0.04074260592460632, 0.731732964515686, 0.8860049247741699, 0.5671272873878479, -1.182747483253479, -0.7637284398078918, -1.2150789499282837, 0.06404846161603928, 0.9949413537979126, 0.7904939651489258, 0.3638865053653717, 1.9467710256576538, -0.0009654454188421369, -1.2820760011672974, -0.74250727891922, -1.1054260730743408, 1.1650972366333008, -0.029407275840640068, -0.4933509826660156, -1.1654012203216553, -0.10590212047100067, -0.7393977046012878, -0.3108924925327301, 1.5745586156845093, -0.5211952924728394, 0.21006067097187042, -0.8196840286254883, -0.5754159092903137, 0.08431075513362885, -1.2085040807724, -0.10596194863319397, 2.9222044944763184, 1.3121421337127686, 1.127195954322815, 0.5231081247329712, 0.6358593106269836, 1.785069465637207, -1.6718649864196777, -0.3736518919467926, -0.6712822318077087, -0.05796971172094345, -0.07800305634737015, 1.9612371921539307, 1.0305830240249634, -1.0629925727844238, 2.31268310546875, -0.17715111374855042, -1.285822868347168, -0.09418027102947235, 0.7899071574211121, 0.115531787276268, -1.641632080078125, -1.9302895069122314, 2.010249137878418], [-0.6522538065910339, -0.16943466663360596, -0.7327204942703247, 0.2112041562795639, 0.09726274013519287, -0.5331650376319885, -0.39857399463653564, 1.9209550619125366, 1.151015281677246, -0.8844631314277649, 0.7063306570053101, -0.533198893070221, 1.6677721738815308, 1.5169661045074463, -0.10169188678264618, 0.02443292923271656, -0.9739500284194946, -0.17668521404266357, -0.6579224467277527, -1.043614149093628, -1.601672887802124, -0.34392502903938293, 0.12690038979053497, -1.8622933626174927, 1.8547863960266113, -0.5209785103797913, 1.0283796787261963, 0.21322448551654816, -1.1170622110366821, -0.9194915294647217, 0.4098207354545593, -0.059307556599378586, 1.4061017036437988, 0.7350631952285767, 0.48450884222984314, 0.1687297374010086, 0.7928224802017212, 0.7105991244316101, 0.3901059329509735, 2.0794312953948975, -0.8897309303283691, 0.1402883380651474, 1.120633602142334, 0.8343303799629211, 0.25222480297088623, 1.1095809936523438, 1.5877782106399536, -0.20023266971111298, 0.26854774355888367, 1.2611627578735352, -0.45903366804122925, -0.6906383633613586, 1.672622561454773, -0.4052915573120117, -0.43794941902160645, -1.8658132553100586, 1.1405956745147705, -0.6957376599311829, -2.4779670238494873, -0.7520730495452881, -0.0622791163623333, 0.4701215624809265, -1.5611523389816284, 0.6748583316802979, 0.11475759744644165, 1.8141894340515137, 0.696165919303894, 1.2681218385696411, -0.2612064480781555, -0.006940032355487347, -0.033901575952768326, -0.7653824090957642, -1.7915594577789307, 0.4503415524959564, 0.38530290126800537, 0.8659111857414246, 1.2290921211242676, -0.23831535875797272, 1.7501094341278076, -1.070114016532898, 0.11541265994310379, 0.09854072332382202, 0.041610874235630035, -0.43464988470077515, -0.5853456258773804, 0.7560644149780273, -1.1068962812423706, -1.5909546613693237, 0.2593856453895569, -1.2580289840698242, 1.2786848545074463, 1.3423457145690918, -0.06408030539751053, -0.25018468499183655, -0.18840545415878296, -0.7018937468528748, -0.4931681752204895, -0.2230161428451538, -0.2279050350189209, -1.589764952659607, -0.4168524742126465, 0.42399418354034424, 1.4426449537277222, -1.5709028244018555, -0.8798632621765137, 0.697615385055542, 0.8885865211486816, -0.32771024107933044, -0.3407801687717438, 0.011963393539190292, 1.6100802421569824, 0.12350554019212723, -0.313086599111557, -1.0802325010299683, 0.9352281093597412, 1.4449743032455444, 0.20708270370960236, -0.7877798676490784, -1.0046762228012085, -0.40663978457450867, 1.1136562824249268, 0.6942866444587708, -0.14708653092384338, -0.3347582221031189, 0.04500102251768112, 1.6271729469299316, -0.1649291217327118, -0.26334771513938904], [-0.8217685222625732, 0.9453756809234619, -0.10805945843458176, 0.6104508638381958, 1.263696551322937, 0.10041876882314682, -0.11887454241514206, 1.331400990486145, -0.7129884362220764, -1.848183274269104, 0.17245064675807953, -1.0567277669906616, 0.3952493965625763, 0.010805203579366207, 1.9369136095046997, -1.026047706604004, 1.7045903205871582, -2.7333619594573975, -0.43433693051338196, -2.507951498031616, 0.03648177161812782, 0.17497535049915314, 0.7347514033317566, -0.34013015031814575, 0.43568962812423706, -0.6859201192855835, -0.9696358442306519, 0.9470031261444092, 1.0935497283935547, 0.588288426399231, -1.0546960830688477, 1.285019040107727, -0.24296869337558746, -0.6180821657180786, 0.11190158128738403, -0.6835626363754272, -1.2808716297149658, -0.9788998365402222, -0.34934452176094055, -1.0250887870788574, -1.0012037754058838, -0.46224460005760193, -0.1471937745809555, 1.2836995124816895, 1.4436243772506714, 1.1424251794815063, -0.9386257529258728, 0.5624061822891235, -0.20890167355537415, 0.42883166670799255, 1.0408813953399658, -0.309980571269989, -1.3193966150283813, -0.9714531302452087, 1.4345026016235352, 0.5545352697372437, -1.3236925601959229, -1.6468826532363892, 2.2580208778381348, 1.4885412454605103, 0.6039423942565918, 0.232587993144989, 0.4305546283721924, -0.1563902050256729, -0.7061384916305542, -0.2537039816379547, -0.9166035056114197, 1.5550827980041504, 1.3992692232131958, 0.9522679448127747, -0.8543938994407654, -0.45486965775489807, -0.24079273641109467, 1.1177120208740234, 0.46873900294303894, -0.11457560211420059, 1.0176560878753662, -0.9184947609901428, -0.5683987736701965, 0.564835250377655, 0.1528194546699524, -2.457206964492798, -1.242165446281433, 1.089112639427185, -0.3433130979537964, -0.0038441673386842012, -1.776776909828186, 0.24629560112953186, 1.60441255569458, 1.4113603830337524, 1.0984551906585693, 0.07685159146785736, -1.260240912437439, 0.35968708992004395, -0.27494052052497864, 1.009390115737915, -0.7517189979553223, 0.9147520065307617, 1.626779317855835, -0.2952643930912018, -0.022953474894165993, -0.4203304350376129, -1.2850561141967773, 1.7706308364868164, 0.48785659670829773, 0.6401118040084839, -0.9915173053741455, 1.8610844612121582, -0.024371765553951263, 1.6462482213974, 1.303301215171814, 1.326924443244934, -0.6454318761825562, 0.364125519990921, -0.1692298948764801, 1.199713945388794, -0.13996060192584991, -0.8569706082344055, 0.14000679552555084, 0.9698678255081177, -0.03778814524412155, 0.47583234310150146, 1.2489734888076782, 0.38767924904823303, -0.48691490292549133, 1.5345808267593384, -0.35030806064605713, 2.0265934467315674], [-0.43761542439460754, -0.014792338013648987, -1.376862645149231, 0.36164045333862305, 1.3657824993133545, -0.511661171913147, 0.4605236351490021, 0.3480198383331299, -0.11118793487548828, -0.8647215962409973, 0.18207699060440063, -0.6838656067848206, 0.8397624492645264, -0.34733012318611145, 0.11714063584804535, -1.6928671598434448, 0.24431119859218597, -0.6675083637237549, -0.6126616597175598, -1.5344080924987793, -0.3664148151874542, -0.7450488805770874, 0.29374000430107117, 0.9906485676765442, 0.08044610917568207, 0.17599739134311676, -0.6226302981376648, -0.06563811749219894, -0.5503016710281372, -0.007428525947034359, -0.1571034938097, -1.9031518697738647, -1.2201343774795532, -1.1933566331863403, -1.6372685432434082, -0.2641684412956238, -1.3367637395858765, 0.4717008173465729, -1.2090405225753784, -0.20499324798583984, -2.2930967807769775, 0.6527910828590393, -2.581070899963379, 0.3191966116428375, -0.6724383234977722, -0.8273330330848694, -0.9469701051712036, -0.795184850692749, 0.027267929166555405, 0.3835744559764862, 0.9731076955795288, -0.33314210176467896, 1.941713809967041, 0.19701670110225677, -0.10511398315429688, -2.07336688041687, 0.02718576416373253, 1.2470197677612305, 0.05373372882604599, 0.9580878019332886, -0.42152339220046997, -1.1129767894744873, -0.5180655717849731, 0.0168134868144989, 1.4140392541885376, 0.3943137526512146, -1.6657429933547974, -0.7566961050033569, -0.9668576717376709, 0.699731171131134, 0.5695874094963074, 0.15433257818222046, -2.35071063041687, -0.35880711674690247, -2.050384998321533, 1.709041953086853, 0.22821295261383057, 0.18293538689613342, 1.0462641716003418, -0.1302899718284607, -0.6169705390930176, 0.6150175333023071, 0.09112253785133362, -2.2503154277801514, 0.5262961387634277, -0.2623142600059509, 1.172029972076416, -0.4050358831882477, 0.6188161969184875, 0.22997719049453735, -1.9033135175704956, 1.5615224838256836, -1.8155019283294678, -1.7669330835342407, 2.1051228046417236, 0.08787012845277786, -0.1084488108754158, 0.1930125504732132, 1.6369630098342896, -0.6845299601554871, 0.10541276633739471, -0.095376156270504, 1.2393698692321777, 1.5559132099151611, -0.20267193019390106, 0.8889117240905762, -1.3498413562774658, -0.222185879945755, -0.8627582788467407, -0.6597787141799927, 0.0628959909081459, 1.2468141317367554, 0.7406957149505615, -1.0595444440841675, -0.44600340723991394, -2.03901743888855, 0.31022992730140686, -0.04639926552772522, -0.5237605571746826, -1.1018129587173462, 0.28639277815818787, 1.173557996749878, 1.2283018827438354, -0.30446577072143555, -0.5221477150917053, 1.788213849067688, 1.1249618530273438, 0.4194752275943756], [-0.41163045167922974, -0.5974428653717041, -0.5434810519218445, -0.15143324434757233, -1.1013965606689453, 0.9470228552818298, 0.13573342561721802, 0.698334813117981, -0.20145033299922943, 1.573253870010376, 1.086297869682312, -0.21653740108013153, -1.3582093715667725, 1.8081178665161133, -1.2509424686431885, -1.6125675439834595, 1.325749397277832, -0.9488911628723145, -0.09785857796669006, 1.6666148900985718, 0.3207959830760956, 1.8240097761154175, 0.28849366307258606, -1.6933902502059937, 0.6343601942062378, -0.4662655293941498, 1.2786451578140259, -0.46451011300086975, -0.2905012369155884, -1.0684475898742676, -0.5836449265480042, 0.08114629238843918, 0.5508427619934082, -0.19753220677375793, -0.6937586069107056, -0.40034034848213196, 0.8873762488365173, -0.004730222746729851, -1.4628850221633911, -0.0854986310005188, -0.8167746067047119, 1.6133062839508057, -0.7226545214653015, 0.503756582736969, -1.4473150968551636, -0.35883212089538574, -1.0387974977493286, -2.7267050743103027, -0.7142855525016785, -0.36701369285583496, -0.7143040299415588, -0.9717468023300171, -0.2072857916355133, 0.5210498571395874, 1.3188802003860474, -0.1323036253452301, -0.2256748527288437, -0.3145330846309662, -1.3333332538604736, -0.5956048965454102, 2.239572763442993, 0.462026983499527, -1.0270825624465942, 0.9939824938774109, -0.2645772397518158, 0.7574008107185364, -0.45309725403785706, -0.36688095331192017, -0.8765560388565063, -0.2661290764808655, 0.5335756540298462, 0.4956183135509491, -0.36990833282470703, -0.4077608585357666, -0.288239061832428, 0.21026071906089783, 0.4088630974292755, -0.28380700945854187, 0.8755524158477783, 0.33206701278686523, 0.1668885350227356, 0.3672214150428772, 1.603805661201477, 0.01734374277293682, 0.1836753636598587, -0.14351843297481537, 1.9480197429656982, 1.1832177639007568, 0.43661293387413025, 0.026486005634069443, 2.6796979904174805, 1.2691636085510254, -0.34138020873069763, 0.44741198420524597, 0.8761164546012878, -0.2154477834701538, 0.14765430986881256, -0.4327271282672882, -2.563267707824707, 0.38919591903686523, 0.09434101730585098, 1.6068147420883179, 0.8097140789031982, 1.1400566101074219, 0.5147802233695984, -0.5477482080459595, 0.6824995279312134, 0.6506572365760803, 0.9687690138816833, -0.23012462258338928, -1.483079195022583, -0.03727182000875473, 0.5226946473121643, -0.2156665325164795, 1.6126251220703125, -1.111256718635559, 0.0024230999406427145, 0.31541094183921814, -1.9033920764923096, 0.5872029662132263, -1.6149650812149048, -0.39917248487472534, -0.35015469789505005, -1.1528180837631226, -1.7907490730285645, 2.316532611846924, -0.37473833560943604, -1.1941235065460205], [0.14660094678401947, -0.9943248629570007, 1.077983021736145, -1.3462501764297485, -1.4859007596969604, 0.03050614334642887, 0.6690276861190796, -0.5053986310958862, -1.8175055980682373, 0.26394209265708923, -0.17666451632976532, -0.49872997403144836, 0.10962381958961487, -1.3414618968963623, -0.8430644273757935, 1.4314911365509033, 0.18091118335723877, 0.25102582573890686, 0.3992370069026947, 0.375230610370636, 0.0011632618261501193, -0.7446199059486389, -2.4559857845306396, 0.15970443189144135, 0.8260374665260315, -1.873087763786316, -0.7812329530715942, 0.033432379364967346, 1.0619608163833618, 1.0058741569519043, 0.38374975323677063, 0.8322835564613342, 0.7215616106987, -0.3557044565677643, -0.7661757469177246, -2.5840044021606445, -0.35330772399902344, 0.5374554991722107, 1.2227717638015747, 0.41320884227752686, -1.437935471534729, -0.19572220742702484, 0.39472153782844543, -0.6782954335212708, -0.5004286766052246, -1.7826812267303467, -0.5405934453010559, -1.254795789718628, 1.2092660665512085, 1.7615541219711304, -0.22771427035331726, -1.403225302696228, 0.213036447763443, -1.1578449010849, 0.8525311350822449, 1.9944111108779907, 1.1898375749588013, -0.6886191368103027, -0.24722355604171753, -0.1825227439403534, -0.8476340770721436, 0.3236966133117676, 0.04526655748486519, -0.28946247696876526, 0.19894354045391083, 1.4058504104614258, -0.2825111150741577, -0.16427059471607208, 0.024816062301397324, 0.552213728427887, 1.120011568069458, -0.1457691192626953, -1.3754873275756836, 0.3120931386947632, 1.1026610136032104, -0.8326013684272766, -0.36840447783470154, -0.4628343880176544, -1.4513579607009888, 0.08439585566520691, 0.3279498219490051, 0.950234591960907, 0.5673049092292786, -1.5149518251419067, -0.5190092921257019, -0.7339840531349182, 1.5389354228973389, -0.009493747726082802, 0.18415892124176025, 1.3696284294128418, -1.8402589559555054, 0.3583970069885254, -1.2462178468704224, -0.5895755887031555, -0.4936516582965851, -0.2546910047531128, 1.2147713899612427, 0.30411195755004883, 0.38059234619140625, -0.11972304433584213, -0.8344109654426575, 1.4749557971954346, 0.2723429501056671, -1.3048248291015625, -1.849221110343933, 0.8933733701705933, -0.9460949301719666, -0.19234158098697662, 0.4819498062133789, 0.16242265701293945, -0.4765366017818451, 0.7359886765480042, -0.12429752200841904, -1.4391682147979736, -1.0313560962677002, -0.3027993142604828, -0.5752386450767517, 0.001837314572185278, -1.1515967845916748, -0.9241822361946106, 1.1906198263168335, 0.7288256287574768, -0.27066054940223694, -0.9208565950393677, -1.4791545867919922, 1.23431396484375, 0.2566356360912323, 0.4110599160194397]]], [[[0.02448185905814171, 0.7677097320556641, 0.26869940757751465, -1.4666343927383423, 1.9175870418548584, 1.1470410823822021, 0.152822807431221, -1.9727234840393066, -0.19732129573822021, 2.2085704803466797, -0.11826065182685852, 0.6673054099082947, -0.9700974225997925, 1.729263186454773, -0.8369856476783752, -1.0565041303634644, 2.484126567840576, -0.0990760549902916, -0.6774198412895203, 1.879263997077942, 0.14911508560180664, -0.9336154460906982, -0.20501790940761566, -1.9578266143798828, 0.03228577598929405, -1.559158205986023, -0.004404398612678051, -1.8468267917633057, -1.9449619054794312, 1.4515190124511719, 0.504618227481842, 0.6184650659561157, 0.7530920505523682, -1.4359194040298462, 0.7730253338813782, -1.2758432626724243, 1.1068278551101685, -0.2436511218547821, 0.7450934052467346, -0.10851166397333145, 2.3700101375579834, 0.7995666861534119, 0.873540997505188, -1.4536727666854858, -0.8156836032867432, 1.2001917362213135, -0.6597049236297607, -1.919579267501831, -1.5316541194915771, -0.2587011158466339, 0.344975084066391, 0.1884002536535263, 0.7742927074432373, 0.7761949896812439, -0.49903908371925354, -1.2268904447555542, 0.1824672371149063, 2.3196470737457275, -1.8283438682556152, -0.6333131790161133, 1.104933261871338, -0.7367445826530457, -2.2096309661865234, 1.934537410736084, -0.3936825096607208, -0.6121600270271301, -0.04794308915734291, -0.8556787371635437, 1.9567739963531494, -1.4465528726577759, -2.693067789077759, 0.45730265974998474, 0.40506047010421753, 0.7989001274108887, -0.7487321496009827, -0.5310860872268677, -0.005169682204723358, 0.00326637364923954, 0.13390837609767914, -0.18229778110980988, -0.267024964094162, 1.239484429359436, 0.5324646830558777, 0.28525474667549133, -0.8302850723266602, 0.12121812254190445, 0.8517482280731201, 0.80357825756073, 0.28388965129852295, 1.884385347366333, 1.0003454685211182, -1.7947735786437988, -2.227803945541382, -0.604864239692688, -1.6965556144714355, 1.7483528852462769, -2.0130221843719482, -0.5012568235397339, 1.5502758026123047, -1.6536643505096436, 2.6653478145599365, 0.8285954594612122, -1.8272652626037598, -0.38949841260910034, -0.1460680514574051, -0.6411862969398499, -0.0842522606253624, -1.2879188060760498, 1.2869062423706055, 0.20751415193080902, -2.484445333480835, -2.1124706268310547, 3.413768768310547, -0.21612393856048584, 0.41099613904953003, 0.39466744661331177, -0.4437045753002167, 1.1963775157928467, 0.7443804144859314, 2.2082929611206055, -1.356050968170166, 0.8141420483589172, -0.6279206275939941, 0.7632437944412231, -1.790385127067566, 1.8243355751037598, 0.5235496759414673, 0.317815363407135], [-0.6476888060569763, 0.20583568513393402, -0.4824412167072296, 0.9329496026039124, -1.0190455913543701, -0.36923864483833313, 1.457061529159546, -0.5910245180130005, -0.23972371220588684, -1.2734668254852295, 1.0263334512710571, -1.2271816730499268, 1.6978275775909424, 1.5343087911605835, -0.04403449222445488, 1.0108144283294678, 0.5544885396957397, -0.527109682559967, -0.33197134733200073, -0.9391604661941528, 2.189840793609619, -0.4159608483314514, 1.3110244274139404, 0.24286538362503052, -0.12075001001358032, -0.3609735071659088, -1.604457139968872, 0.4096547067165375, -0.4593172073364258, -0.4878008961677551, -0.4776344299316406, 0.4086533486843109, 0.5276384949684143, 1.471174955368042, -0.1808050125837326, -0.47031018137931824, 1.1040666103363037, -1.7539860010147095, 1.4561145305633545, 0.07228093594312668, 0.3337249755859375, -1.8139485120773315, 0.20832936465740204, -0.10571163147687912, -0.8625559210777283, 1.150819182395935, -0.35216742753982544, -1.311265230178833, 0.20800241827964783, 1.726298213005066, -0.6199381947517395, -0.6057175993919373, -0.010347818024456501, 1.064064621925354, 0.7738332152366638, -1.2469494342803955, -0.528974711894989, -1.1401748657226562, 0.6498213410377502, -0.7022775411605835, 0.0885644182562828, -0.03675062954425812, 0.8241100907325745, 0.2637859582901001, -0.1307871788740158, 0.9160415530204773, -0.921876847743988, -0.90997713804245, -0.6318694353103638, -1.033129096031189, 0.6136009097099304, 0.9343653321266174, -0.522525429725647, 0.17260988056659698, -1.3886620998382568, 0.3242170214653015, -0.4710122346878052, -0.7772091627120972, 0.012462170794606209, 0.2187185287475586, -0.017706453800201416, -0.24035967886447906, 0.6014532446861267, -1.331370234489441, 0.8812956809997559, -0.606950044631958, 0.10951083898544312, 0.9038297533988953, -1.2388027906417847, 0.8889403939247131, -0.5162791013717651, -0.1078273355960846, -1.0367401838302612, -0.23330016434192657, 0.5871246457099915, -1.5018866062164307, -0.32162266969680786, 0.537152111530304, 0.19899792969226837, -0.8965430855751038, -0.13917571306228638, -0.6746553778648376, 0.3081928491592407, 1.2764663696289062, 0.11549052596092224, -1.664250373840332, 0.37249529361724854, 0.2634139955043793, -1.3273674249649048, 2.063229560852051, -1.01571786403656, -1.9555250406265259, -1.1526318788528442, -0.5696362853050232, -0.18210245668888092, 0.600472092628479, 0.028232352808117867, -0.5800712704658508, -0.7277379631996155, 0.18438895046710968, -1.7765709161758423, 0.34426918625831604, -0.1037227213382721, 0.18440841138362885, 0.5416735410690308, 0.46816790103912354, 0.09317536652088165, -1.090173363685608], [0.9128084778785706, 0.17644083499908447, 0.3414241075515747, 1.3949897289276123, -0.09916940331459045, -0.30561718344688416, 0.9318339824676514, -0.515544056892395, -0.19161419570446014, 0.6006439328193665, 1.650509238243103, 1.629753828048706, 2.0188658237457275, 0.8094847202301025, 0.5447234511375427, -1.991302728652954, 0.4988342821598053, -0.25789812207221985, 1.1249468326568604, -0.16985104978084564, 1.3428064584732056, -2.3913094997406006, -1.8026180267333984, 0.4746532142162323, -1.132466197013855, -1.1170734167099, -0.6996747851371765, -0.1058412417769432, -0.7789146304130554, -1.0617740154266357, -0.9793086051940918, -0.0280990581959486, 0.8647139668464661, -0.12088757753372192, -0.8887237310409546, -1.130652904510498, 0.13536466658115387, 1.5329679250717163, 0.15823876857757568, 2.0400733947753906, -0.9097766876220703, 3.228903293609619, 0.9031848907470703, 0.006323200650513172, 1.5150010585784912, 0.8985671997070312, 0.30887725949287415, 1.0561765432357788, 0.40208739042282104, 0.6917154788970947, 1.7193281650543213, -0.3414320647716522, -0.6449424624443054, 0.6726080775260925, -1.0639504194259644, 0.761633574962616, 1.784532070159912, -0.5552530288696289, 0.03206481784582138, -1.221671223640442, -0.13775630295276642, -1.0888679027557373, 3.303589105606079, 0.3665515184402466, 1.1761595010757446, 0.22777439653873444, -1.2483007907867432, -0.03696794807910919, -0.08794622123241425, 0.3642462491989136, -0.8514370918273926, 0.029666727408766747, -0.3208983242511749, 0.08582472056150436, -0.742844820022583, 0.0044226376339793205, 0.6905057430267334, -0.902776300907135, 2.83182430267334, -0.05825558677315712, 0.10719742625951767, -0.5116783976554871, 0.29624009132385254, 1.397982120513916, 1.2376877069473267, -1.5993273258209229, 0.5331620573997498, -1.2683157920837402, 0.26179662346839905, 0.5388706922531128, 0.8144378066062927, 0.6608511209487915, -0.8398016691207886, 0.1560155600309372, 1.229217290878296, 0.3182811141014099, 0.4879171848297119, -0.2406652867794037, 0.08076322078704834, 0.7434536814689636, -0.3174993693828583, 0.1388251930475235, 0.4337250292301178, 1.1729881763458252, -0.592949628829956, 1.447239875793457, -0.24084384739398956, -1.782473087310791, -0.06203274801373482, 0.30960121750831604, 0.4792580008506775, 0.5233916640281677, 0.7247862815856934, 0.8224000334739685, -0.17614249885082245, 0.10105282068252563, 1.1182689666748047, -0.9819470643997192, -1.918686032295227, -0.6945884823799133, 0.04341096803545952, 0.29337361454963684, 1.9643968343734741, 1.4123435020446777, 0.8859716653823853, 0.737770676612854, -0.469499796628952, -1.7575591802597046], [0.6765257716178894, 0.183353453874588, -0.5500777363777161, 1.4439027309417725, 1.6711981296539307, 0.5853780508041382, -0.33381521701812744, -1.4294648170471191, 0.023982945829629898, -0.9132371544837952, 1.0903552770614624, 1.1916530132293701, 0.09672325849533081, 0.1348971724510193, 0.6726164817810059, -0.26156163215637207, -1.114250659942627, 1.4083659648895264, 0.3697601854801178, -1.8868191242218018, -0.8045341372489929, 2.405689239501953, 0.13483141362667084, 1.168255090713501, -0.14949320256710052, -0.5269834399223328, 0.5856959819793701, 2.44640851020813, -0.6483399868011475, -1.0697005987167358, -0.021027056500315666, 0.9336798191070557, -0.6348645687103271, -0.026622407138347626, -0.8249268531799316, -0.9021540880203247, 0.22492378950119019, -0.4350849390029907, -0.3352920413017273, -0.42687785625457764, -1.0761276483535767, -1.5404889583587646, 0.7090396285057068, -0.9667379260063171, 1.264998435974121, -0.1400068998336792, 1.1341263055801392, -0.7854750156402588, -0.5871216654777527, -1.034458875656128, -0.8360263109207153, 1.8778631687164307, -0.47063663601875305, 1.0226033926010132, 0.876922070980072, 1.4012534618377686, -0.5696443319320679, -0.28219908475875854, 1.678260087966919, -0.13849714398384094, 0.0472259521484375, 0.48374202847480774, -2.9499526023864746, 0.8053758144378662, 2.665315628051758, -0.7567940354347229, -0.5513619184494019, 0.6166907548904419, 0.44282281398773193, 0.5104397535324097, -0.9740613102912903, -0.7256649732589722, 1.663432002067566, -1.7516648769378662, -0.7113449573516846, -0.30955958366394043, -0.5234830379486084, 0.4464018940925598, -0.8339254856109619, -0.5951151847839355, -0.2000766396522522, -0.2835247814655304, -0.13717491924762726, 1.2601919174194336, 1.5229759216308594, 2.5322697162628174, 0.5784055590629578, 1.5982325077056885, 0.13979466259479523, 0.35138577222824097, -0.4868674576282501, -0.9821734428405762, -0.5790908932685852, 0.31319594383239746, -1.203905701637268, 1.422532320022583, -4.820175647735596, 0.16138969361782074, -0.7456915974617004, -0.9702907204627991, 0.16261029243469238, 0.4561690092086792, -0.08138522505760193, 0.2690717875957489, 0.16383200883865356, 0.6691933274269104, -0.12873294949531555, -0.11867298185825348, 0.11793386936187744, -1.9027374982833862, -1.8959755897521973, 0.1732272505760193, 0.22842136025428772, 1.2057465314865112, -0.9998956322669983, -0.31799861788749695, 0.8704087734222412, -0.7780575752258301, 1.0583184957504272, 0.03292539715766907, 0.35230395197868347, -0.4762890934944153, -0.31279081106185913, 0.6780009865760803, 0.9783734083175659, 0.24701239168643951, -0.46862971782684326, 0.6859331727027893], [0.745937705039978, 0.02526259422302246, 1.126860499382019, 0.7698380947113037, 1.0042908191680908, 1.054681658744812, 0.09062433242797852, 1.0338680744171143, -0.03737514466047287, 1.2435319423675537, -0.7686936855316162, -0.02897353656589985, -0.5655792355537415, -0.8254125118255615, -3.2538578510284424, -0.9616493582725525, -1.5559948682785034, -0.160612091422081, -1.4414799213409424, 1.515695333480835, -0.12992356717586517, 0.8789687156677246, -0.6993891000747681, 0.01329335942864418, -0.008344980888068676, -0.6041834354400635, 1.4935340881347656, 0.46901440620422363, -0.3323078155517578, 0.5094171762466431, 0.14115135371685028, -0.8608770966529846, 2.0694403648376465, 1.8251607418060303, 1.8465538024902344, -0.06382500380277634, -1.7833517789840698, 0.9265088438987732, -0.5580200552940369, -1.3221054077148438, 0.05983983352780342, 0.5509008169174194, -0.011265551671385765, -1.3969316482543945, -0.4037136137485504, -0.12961626052856445, 0.5229390263557434, 0.6187868714332581, 0.8326910734176636, -0.41101714968681335, -0.22406595945358276, -0.08056062459945679, -1.9480705261230469, -0.7482137680053711, -0.562956690788269, 0.3778020739555359, -0.02460491470992565, -1.9563674926757812, -0.383686900138855, 0.8866170048713684, -0.3839203119277954, 0.8167117238044739, 1.3645858764648438, 0.9405843615531921, 0.6217637658119202, 0.5243932604789734, 1.0919774770736694, -0.11330369114875793, -0.7856388092041016, 0.14367996156215668, 0.4706508219242096, 0.043990593403577805, 0.14945384860038757, 0.12016648799180984, -0.34977519512176514, -0.3841700255870819, -0.4797309637069702, -1.9085888862609863, 2.4058310985565186, 1.1312024593353271, -0.9974510669708252, -0.9464389085769653, -0.07238688319921494, 0.2882467210292816, -0.5237963795661926, 0.8887166380882263, 0.5898932814598083, 0.061291761696338654, -0.45149022340774536, 1.4956358671188354, -2.5721144676208496, 1.6086556911468506, -0.5446403622627258, -1.5192233324050903, -0.7116296887397766, 1.331295132637024, 0.8295690417289734, -1.184801697731018, -1.5039576292037964, -0.7995752096176147, 0.4139895439147949, 0.05239930748939514, -0.4119311273097992, -0.9282578229904175, 0.612138032913208, 1.6187089681625366, 0.8177744746208191, 2.9425878524780273, -0.6066650152206421, 1.024686336517334, -0.5676119923591614, 1.3843895196914673, -0.017566276714205742, 0.8546543717384338, -0.41224735975265503, 0.7038244009017944, -1.0565388202667236, -0.6482229232788086, -1.6920634508132935, -0.6247544884681702, 1.5892177820205688, -0.8576380014419556, -1.062958836555481, -0.7235020995140076, -0.34170836210250854, 1.9726510047912598, 0.2369910627603531, 1.1269488334655762], [0.44871073961257935, 0.21648700535297394, 1.5183604955673218, -0.8659560680389404, 1.7301867008209229, -0.7239362597465515, -0.6477221846580505, 1.407391905784607, -0.5969987511634827, -0.5912616848945618, 1.3485054969787598, -1.739894151687622, -1.3390944004058838, 1.1706682443618774, 0.5401647090911865, 1.9267566204071045, 2.4354867935180664, -0.8288998603820801, -0.6729724407196045, -1.0538939237594604, -0.7806921601295471, 0.6147233843803406, 0.4619240462779999, -0.7668566107749939, -0.6237510442733765, -1.1732409000396729, 0.3684265911579132, -1.5276775360107422, -0.838951826095581, -2.2637147903442383, -1.4296627044677734, -1.7095342874526978, 1.4447447061538696, -2.322061061859131, -0.5051980018615723, -0.32768714427948, -0.19208814203739166, 0.5397306680679321, 0.21156789362430573, 0.8232102394104004, -0.947512149810791, -0.0934741348028183, 1.7940967082977295, -0.05474310368299484, 0.09241490811109543, -0.9917853474617004, -1.3041669130325317, 0.44936898350715637, 0.7103468775749207, 0.9141449928283691, 0.9358986020088196, -0.6025363802909851, 0.422551691532135, -0.4311244487762451, -0.045242276042699814, -2.7442378997802734, 2.0200846195220947, 1.1067787408828735, 1.4601936340332031, -0.042931508272886276, -0.8528750538825989, -0.8644986152648926, 2.5943846702575684, 0.5146707892417908, -1.9018014669418335, -0.17117075622081757, -0.4492552578449249, 0.4912320375442505, 1.912423849105835, 1.2212145328521729, -0.45380502939224243, -0.08923421055078506, 0.20546500384807587, 1.006488561630249, 0.5090993642807007, -0.05160904675722122, -1.1840710639953613, 0.9803245663642883, 1.9649810791015625, -0.10011991858482361, -1.6843292713165283, 0.2905728816986084, 0.6310686469078064, 0.814713180065155, 0.9034900665283203, -0.06836636364459991, -1.2676876783370972, 1.0957138538360596, 0.33407580852508545, 1.1805758476257324, -0.853001058101654, -1.3942091464996338, 0.6333721280097961, 0.272551566362381, 0.6152641177177429, -0.4567748010158539, 1.3012579679489136, -0.3444361090660095, 1.1484854221343994, 0.8160215020179749, 0.5625500679016113, 1.4831291437149048, -0.06800162047147751, -0.5173318386077881, 0.45304709672927856, 1.6249141693115234, -0.09626076370477676, -0.5118141174316406, 0.008702577091753483, 0.9286964535713196, 0.42866089940071106, -1.7336794137954712, -1.5415457487106323, 0.7385653257369995, 0.0979432463645935, 1.650902509689331, -0.2787715792655945, 0.8551592826843262, 0.037656791508197784, -1.325850009918213, 0.7064269185066223, 0.20846788585186005, -0.05205410718917847, 1.2698440551757812, -0.5671555399894714, -0.01875930279493332, -0.286900132894516, 0.40904247760772705], [1.340644359588623, -1.355794072151184, -0.027303487062454224, -0.3982454538345337, -1.380478024482727, -0.08122152090072632, 2.6758430004119873, 0.5148954391479492, -0.2396203726530075, 0.5979204773902893, -1.6229151487350464, 0.4698674976825714, -1.9427390098571777, -1.286747932434082, -0.1723901927471161, -0.21629168093204498, 0.6797260642051697, 1.5572375059127808, -1.2500981092453003, -0.4107741117477417, -0.986835241317749, 0.3189694285392761, 1.1828124523162842, 0.2671050429344177, 1.7412410974502563, -0.17093078792095184, 0.3099488914012909, 0.3942311108112335, 0.5864620208740234, -0.568083643913269, -0.3637349307537079, 0.3482023775577545, 1.7247858047485352, 1.4142144918441772, 1.3860208988189697, -0.39428701996803284, 0.5237748622894287, -0.03294077888131142, 0.303300142288208, -0.7740530967712402, 0.41951626539230347, -0.45312637090682983, 0.5112583637237549, 0.9699077606201172, 0.8672553300857544, 0.4201951324939728, 1.734157681465149, -1.673262596130371, 0.08294888585805893, 1.4854841232299805, 0.5016087293624878, 1.7039172649383545, 0.946838915348053, 2.055453062057495, 0.0060079945251345634, -0.04395897313952446, -1.2356970310211182, -2.0640676021575928, -0.2651773989200592, 0.7348682284355164, 0.6720324754714966, -0.5884284973144531, -1.333330750465393, -0.11707667261362076, 1.0757766962051392, -0.8601413369178772, 0.49627405405044556, 1.1184402704238892, -0.8770818710327148, -0.3847719132900238, 2.0068724155426025, -0.21996307373046875, 0.4969042241573334, -0.8723759055137634, -0.7649458646774292, -0.4657686650753021, -1.2244489192962646, 1.2660642862319946, 2.420969247817993, -1.3764972686767578, 0.9573250412940979, 0.06681910902261734, -0.2471386045217514, -0.5816606879234314, 1.2072789669036865, -0.8397735357284546, 1.141706943511963, 0.8679011464118958, -2.30021071434021, -0.6142427325248718, 0.8078945875167847, -0.9551203846931458, -1.3604480028152466, -0.4405185878276825, -1.1037124395370483, 0.6524376273155212, -0.45613232254981995, -2.5641586780548096, -0.14324405789375305, 1.6839944124221802, 0.031356073915958405, -0.7211329340934753, -0.7097090482711792, 1.2916525602340698, 0.01208052784204483, -1.138242483139038, -0.09142931550741196, 0.28319084644317627, 1.3706121444702148, 1.7281602621078491, -1.1340057849884033, -0.8476801514625549, 1.4307681322097778, 0.8393747210502625, -0.7048782706260681, 0.8324342370033264, 0.3911930024623871, -1.3326040506362915, 0.1965458244085312, 0.9107168912887573, 0.7961984872817993, -0.1502142995595932, -1.290765404701233, -0.0614108182489872, -1.3402621746063232, 1.1443514823913574, 0.9404212236404419, 1.7248727083206177], [-1.0560365915298462, -0.4730237126350403, 0.7768740653991699, -0.46603405475616455, -2.127021551132202, 0.5573250651359558, 1.386657953262329, 0.0755394771695137, 0.0646592378616333, -0.01578443869948387, 1.353442668914795, -0.6549813747406006, -0.011421403847634792, -1.6865023374557495, -0.5455114841461182, 0.9014130234718323, 1.242668867111206, 1.0723968744277954, 1.4599674940109253, 1.4053857326507568, 0.4964004456996918, 0.04070070758461952, 0.7309662103652954, 0.9052167534828186, 2.524170160293579, 0.2973141372203827, -0.5990496277809143, 1.1493607759475708, 0.28896862268447876, -0.7028151750564575, -2.902761936187744, 0.8703272938728333, -0.1294102966785431, -0.037658143788576126, 0.5453738570213318, 1.7811055183410645, 0.36840271949768066, 0.2454691231250763, -0.41512539982795715, 0.8465649485588074, -0.666235625743866, 1.6222729682922363, 0.01841888576745987, 0.4527198076248169, 0.6736987829208374, -1.0766502618789673, -2.1340079307556152, 0.3654583692550659, 0.43920251727104187, 0.015577276237308979, 0.5466739535331726, 0.20866678655147552, -0.9172056913375854, 0.11838113516569138, -0.36402758955955505, -2.175936698913574, -0.530074954032898, 1.6198822259902954, -0.559438169002533, -1.2084085941314697, -0.04710075259208679, 0.259878933429718, -0.5267049670219421, 0.034638624638319016, 1.4160099029541016, -0.3909182846546173, -3.184718132019043, -1.4879939556121826, -1.4497522115707397, -0.3492499589920044, 1.1294407844543457, 1.1189380884170532, 0.24853552877902985, 1.2973936796188354, -0.005854939576238394, -0.6024107933044434, 1.7496553659439087, -1.6561769247055054, 0.2071857452392578, 1.343438982963562, 0.6042656302452087, 1.610970139503479, -0.5428681373596191, 1.4778789281845093, 0.859003484249115, 0.8280092477798462, -0.09811946004629135, -0.925775945186615, -1.1605912446975708, -0.8658491373062134, -0.603946328163147, 0.36895060539245605, 0.6842219829559326, 0.6598710417747498, -0.13912923634052277, 1.4343727827072144, -1.3246564865112305, -1.5401781797409058, -0.5036640763282776, 0.8316934108734131, -1.3039124011993408, 0.516282856464386, 0.9734742045402527, 1.3343952894210815, 0.63005131483078, 0.3685685992240906, -0.4693536162376404, -1.0003958940505981, -1.4009902477264404, 0.09581782668828964, 1.9824960231781006, 1.191725492477417, -1.240726351737976, 1.8429778814315796, 0.1735592931509018, 1.6166725158691406, -1.2886782884597778, 0.5398942232131958, -0.437678724527359, -0.5283066630363464, 0.3885490894317627, 0.35837996006011963, 0.011783872731029987, -0.15623153746128082, -1.4417495727539062, 0.9379849433898926, -0.47243696451187134, 0.714862585067749]], [[1.9923717975616455, 0.7983180284500122, 0.5925917029380798, 0.6653943657875061, 0.2743847668170929, 0.5733282566070557, -1.2200273275375366, 0.11895608901977539, 0.21299999952316284, -0.7409311532974243, 0.5789064168930054, 0.2927126884460449, 0.6206923723220825, 1.3821194171905518, -0.9716208577156067, -0.05422770977020264, -0.4475657045841217, 0.22486789524555206, -0.08205126971006393, 0.18673300743103027, 1.0624854564666748, 0.23705297708511353, 1.5052417516708374, -0.13933324813842773, -0.20901598036289215, -0.7197533249855042, 1.0468239784240723, 1.0138137340545654, -0.05345411226153374, 0.7927621006965637, 0.15461871027946472, -0.4387294054031372, 1.4790806770324707, -0.11084813624620438, -0.578427255153656, 0.8587244153022766, 1.7287145853042603, -1.1012414693832397, 0.7822652459144592, 0.20326267182826996, 3.4748146533966064, -0.022024381905794144, -0.15434904396533966, 0.8774604797363281, -1.8655989170074463, 0.2728821337223053, 1.2904311418533325, -1.5655338764190674, -0.3661181628704071, 0.041544996201992035, -0.42720475792884827, 1.3755035400390625, 1.1955970525741577, 0.6390146017074585, -0.8184658288955688, 1.7927478551864624, -1.8814870119094849, -0.29634079337120056, -0.8462123870849609, -0.0676708072423935, -1.0723482370376587, 0.40866291522979736, -1.2097395658493042, 0.4266221523284912, -0.3355658948421478, 0.11175768822431564, 0.32929620146751404, 1.3938241004943848, -1.8521534204483032, -1.3984434604644775, -0.04713795706629753, 0.35686516761779785, 0.45746544003486633, 1.6806477308273315, 0.1345389485359192, 1.0693936347961426, 0.8354633450508118, 1.2422128915786743, -0.11632636189460754, -0.5163020491600037, 0.5529488325119019, 0.981134295463562, 0.30810871720314026, -0.1780373752117157, 0.47010570764541626, 0.867129921913147, -0.4292084276676178, 1.3400534391403198, 2.053988456726074, 0.2974611818790436, -0.99294114112854, -0.046147529035806656, 0.05433515086770058, 1.544084906578064, -0.5397338271141052, 1.031071424484253, 0.10766074061393738, -0.5201870203018188, 0.8314236402511597, 1.5171711444854736, 1.7832767963409424, 0.5826530456542969, 0.7687479257583618, -0.43186265230178833, 1.250603437423706, 2.261997938156128, -1.9206944704055786, -0.04098997265100479, -1.230965256690979, 1.6962549686431885, -2.311513900756836, -0.3189079463481903, 1.2417829036712646, -0.6582978963851929, 1.7919716835021973, 1.0631954669952393, 0.5159764289855957, 0.5727049708366394, -0.3425382375717163, 0.23475171625614166, 0.2547052502632141, -0.01394769735634327, 0.8729797005653381, -1.424687385559082, 1.3766827583312988, -0.8679646849632263, -2.7549867630004883, 0.4183976352214813], [0.33528363704681396, 0.047793351113796234, 0.5136117935180664, 0.6353681683540344, 0.34559857845306396, -0.8504624962806702, 0.0830734595656395, 1.2580513954162598, 0.8527936339378357, 3.0422279834747314, -1.6874442100524902, 0.17994697391986847, -0.4295074939727783, 1.471693754196167, 1.539783000946045, 0.9046091437339783, 0.353059321641922, -0.47473451495170593, -1.2584929466247559, -0.4661886990070343, 1.2343276739120483, -0.5839711427688599, 0.6592716574668884, -1.099979043006897, -1.3494341373443604, -1.5763561725616455, 0.4159323573112488, -2.3725967407226562, -0.3832431435585022, -0.762843132019043, 0.8043279647827148, -1.555602788925171, -2.0964138507843018, 1.5037305355072021, 0.7000661492347717, 1.4404451847076416, -0.5045846104621887, -0.35357287526130676, 0.20900669693946838, -0.4769305884838104, 0.8355482816696167, -0.5385423302650452, 1.6243267059326172, -0.031613271683454514, -0.5377919673919678, -0.13659648597240448, -0.1707560420036316, -1.4420243501663208, 0.21790294349193573, 0.6686450839042664, 0.04008670523762703, -1.1724389791488647, -1.6198275089263916, -0.30293720960617065, -0.9678935408592224, -0.8439143896102905, -2.197222948074341, -0.13172070682048798, -1.057737946510315, -0.7226272225379944, 0.7893511056900024, -0.411845326423645, -0.2956972122192383, 0.313845157623291, -2.8456664085388184, 1.1974835395812988, -0.34913522005081177, -0.21449467539787292, 1.3213680982589722, -1.3267415761947632, 1.305187463760376, -0.2735164165496826, 1.23294198513031, -0.3095664381980896, -0.4156959652900696, 0.339409202337265, -1.0522536039352417, -0.7258356213569641, 1.1433011293411255, 0.43972957134246826, 1.5505485534667969, -0.5169699788093567, 1.471024513244629, -1.412643313407898, -1.4770252704620361, 0.6595693826675415, -0.7749794125556946, 0.6785833239555359, 1.1385465860366821, 1.4135833978652954, 0.48778852820396423, -0.15984757244586945, 1.1961891651153564, -0.20351780951023102, 0.5475124716758728, -0.7009292244911194, -1.1351661682128906, -0.17250923812389374, -0.006600753869861364, -1.6952470541000366, 0.5012947916984558, 0.7269221544265747, -0.8671849966049194, -0.2231549322605133, 0.13462069630622864, -0.9543787837028503, 0.32038626074790955, 0.07882005721330643, 1.1091145277023315, -0.3286665081977844, 0.4821489453315735, -0.38551566004753113, 0.35328254103660583, 0.4560815095901489, 1.417763113975525, -0.27125072479248047, -0.2994183301925659, 2.5846176147460938, 0.3108341097831726, 1.786159873008728, -0.009537240490317345, -0.3916361629962921, 0.4712553024291992, 1.2440578937530518, -0.4646663963794708, -1.315738320350647, 2.226840019226074, 0.282587468624115], [-0.880725085735321, 0.34459638595581055, 0.9919037818908691, 0.0798708125948906, -1.435280680656433, 0.19128084182739258, 0.262112021446228, -0.8475600481033325, 1.6327807903289795, -1.5776993036270142, 1.4583508968353271, 1.2160255908966064, 1.0182323455810547, 1.7072316408157349, -1.4611949920654297, -0.8312875628471375, 1.344992756843567, -0.4623669683933258, -1.652920126914978, 0.46039828658103943, -0.49249380826950073, -0.7727670669555664, 0.5636411309242249, -1.0980561971664429, 0.12662723660469055, -0.4309983253479004, -0.0733838826417923, -0.2852036654949188, 0.02055881731212139, -0.8984483480453491, 0.2906230688095093, 0.6092696189880371, 0.463331013917923, -0.26131975650787354, -0.1889367550611496, -3.296269416809082, 0.3200463056564331, -1.5550402402877808, 0.27143406867980957, -0.4426671266555786, -0.2471175491809845, 1.4525034427642822, -0.33267736434936523, 0.7299202084541321, -0.21056875586509705, -0.0021759173832833767, -0.29399290680885315, 0.0071929702535271645, 0.8009925484657288, -0.5348021984100342, 0.4376990795135498, 0.5527145862579346, -1.2207152843475342, 0.4461286664009094, -0.7682747840881348, 0.13197383284568787, 0.6816995739936829, -1.8715832233428955, -1.6308971643447876, 1.4155921936035156, 1.4039461612701416, 1.3717013597488403, 0.5837738513946533, -1.4950120449066162, -1.9050837755203247, -1.661252737045288, 0.8930780291557312, -0.2559491693973541, -0.42065364122390747, 0.8269228339195251, -0.47699394822120667, 0.051318854093551636, 0.27660489082336426, 0.1652233749628067, -0.7747721672058105, 0.6435398459434509, -0.014987933449447155, 1.309999704360962, -1.4325149059295654, -0.35889512300491333, -0.16745366156101227, -0.6089552640914917, 0.6632524132728577, -1.0306239128112793, -1.6702498197555542, -0.5365027785301208, 0.6556593179702759, -1.001185655593872, -0.021506115794181824, 0.7531833052635193, -2.7631328105926514, 0.0711163729429245, 0.5860000252723694, -0.8575341105461121, 0.18749703466892242, -1.1681491136550903, 0.32701990008354187, -0.6401777267456055, 0.8989960551261902, -1.4772415161132812, -2.222511053085327, 0.18576034903526306, -0.5477967858314514, -0.1580718606710434, 0.9491028785705566, 0.14375144243240356, -1.2635120153427124, -0.023001575842499733, -1.073399305343628, -0.3580402731895447, -1.5129845142364502, 0.5308614373207092, 0.3248094320297241, 0.2264041155576706, 0.4677886962890625, 0.7231842279434204, 0.12660394608974457, 1.7963590621948242, 1.9443466663360596, 0.885473370552063, 0.9455410838127136, -1.4032819271087646, 1.0132029056549072, -1.2785382270812988, 1.2870244979858398, -0.16818922758102417, 0.2744351625442505, -0.3173532783985138], [-0.7192097306251526, 0.06485041230916977, -0.948416531085968, 0.14880506694316864, 2.3506367206573486, -0.6625604033470154, -0.35424843430519104, 0.09027887135744095, 0.2516613304615021, -0.9209667444229126, 1.3441728353500366, -0.10774378478527069, 1.0739787817001343, -0.08645326644182205, 0.6840302348136902, -0.7563583850860596, 0.6987307071685791, -0.4401525855064392, -0.44016391038894653, -1.1690884828567505, -1.8094477653503418, 2.225067615509033, 0.28656259179115295, 0.3750441372394562, 0.6322923898696899, -1.7460485696792603, -0.4462166130542755, -0.9854358434677124, 0.18348418176174164, -0.15811720490455627, -1.005250096321106, 0.945395827293396, -0.09356493502855301, 0.47707244753837585, 0.2276225984096527, -0.19578424096107483, -0.7276831269264221, -0.25283288955688477, -0.004343811422586441, -0.6074845194816589, -0.10086548328399658, 0.4537847340106964, -0.8660979270935059, -0.8022370338439941, -0.7643497586250305, 1.3569982051849365, -0.5405898094177246, -1.1839263439178467, 0.7262800931930542, -2.014338970184326, 0.11351785808801651, -1.920279860496521, 0.39430883526802063, 0.6098207831382751, -1.4631168842315674, 0.2094542384147644, 0.0964709222316742, 1.6636682748794556, -1.2955491542816162, 1.4059070348739624, -0.2476608157157898, -0.45872169733047485, -0.7514818906784058, -1.7165645360946655, 2.737445116043091, 0.6175253987312317, 1.4766428470611572, -0.9460335373878479, -0.1251029670238495, -1.7387546300888062, 0.06854679435491562, 1.1518433094024658, 0.02943960763514042, 1.4872758388519287, -0.6164153814315796, -0.548712432384491, 0.21091009676456451, 0.4465300142765045, -0.7978692054748535, -0.21875537931919098, 1.2931872606277466, 0.6090320944786072, -0.31008660793304443, 1.2693381309509277, 0.6849637031555176, -0.11772213131189346, -0.037101782858371735, -0.3779795169830322, -0.4421440064907074, 0.687267541885376, -1.6790494918823242, 0.5950948596000671, -0.3208921551704407, -0.5370891094207764, -0.5903297066688538, -0.22617116570472717, 1.076697826385498, 0.2778981626033783, -0.5476650595664978, 0.11938177049160004, 0.835579514503479, 0.5460296869277954, -1.8525127172470093, -0.49513140320777893, -1.4032375812530518, 0.36054733395576477, 1.4782230854034424, 1.3861441612243652, 0.4378153383731842, -0.4771062731742859, 2.337143898010254, -0.4695417582988739, -0.812250554561615, -2.0102739334106445, -0.9899458885192871, -2.4080657958984375, 0.3112933039665222, -1.5683135986328125, 0.0928293988108635, 0.6555100083351135, -0.7589622735977173, -0.35117146372795105, 1.347668170928955, -0.31909582018852234, -0.6307259798049927, -1.0751338005065918, 0.2500764727592468, 0.6716048717498779], [-1.3114763498306274, -0.9545655250549316, -0.962153971195221, 0.9868611097335815, 2.0527403354644775, -0.7147455811500549, 0.9471287131309509, 1.1730138063430786, -1.0999207496643066, 1.5268750190734863, -1.2619373798370361, -1.1691455841064453, -0.535643458366394, 1.4136865139007568, 1.3734627962112427, 0.783497154712677, 1.408657193183899, 0.2853085398674011, -1.2079294919967651, -0.4913545846939087, 0.3572171628475189, 0.23780004680156708, -0.8709983229637146, -0.9822015762329102, -1.2392090559005737, -0.6455107927322388, 0.17953623831272125, -1.9151731729507446, -0.7725074887275696, -0.39041826128959656, -0.6481088995933533, 0.15682554244995117, -0.15366418659687042, -1.783686876296997, 1.4437464475631714, -1.2781496047973633, -0.6757705211639404, -1.1552150249481201, -0.2823825478553772, 0.5631542205810547, -1.4248305559158325, -0.6727563142776489, -1.115440845489502, -1.5503581762313843, -0.43201780319213867, -2.1428353786468506, 1.1054373979568481, 0.14072264730930328, 0.4662769138813019, -0.07662920653820038, 0.025871187448501587, -0.7764281630516052, -1.337964415550232, -1.9752694368362427, -0.7247486114501953, -0.6954493522644043, -0.3440472483634949, 0.42873236536979675, 1.298520565032959, 1.780366063117981, -0.1732066571712494, -0.06129538640379906, 0.2786940038204193, 0.6892370581626892, -1.7463035583496094, -0.08476225286722183, -1.2199450731277466, 0.5986002683639526, 0.12791596353054047, 1.40630042552948, 1.3987154960632324, 1.5094528198242188, -0.4896574020385742, -0.8984569311141968, 1.7970695495605469, -1.7723942995071411, -1.1889787912368774, -0.8819541931152344, 0.4315066635608673, 1.0036112070083618, 1.6185827255249023, -0.6299558281898499, -0.7694566249847412, 0.9896828532218933, -0.4447917342185974, 0.4595760405063629, -0.22570642828941345, -0.849991500377655, -0.5953689217567444, -0.8251193761825562, 0.30984506011009216, 0.1374569833278656, -0.1939551830291748, 0.40308839082717896, 0.7350822687149048, 0.33390966057777405, 0.5476318001747131, 0.6320378184318542, -0.826095700263977, 0.08147622644901276, -1.4708315134048462, -1.0108989477157593, -1.3359031677246094, -0.7151052355766296, -0.4238924980163574, -0.5831649303436279, -0.3896857798099518, -0.2569648325443268, 0.22922749817371368, -0.1311822384595871, 2.4144399166107178, 0.08795411139726639, 0.8136783242225647, 0.021664926782250404, 0.2463984489440918, -0.0072837891057133675, 1.29989755153656, -2.6729986667633057, 1.1500442028045654, 0.2958393692970276, -0.44115495681762695, 1.9204986095428467, -2.494903087615967, 0.9936369061470032, -0.4084477722644806, 0.08203588426113129, 1.0288984775543213, -0.15677061676979065], [1.2542657852172852, -1.1314655542373657, 1.4904589653015137, 0.9183121919631958, -1.774707317352295, 1.7859748601913452, -0.3905893862247467, 0.17084889113903046, 0.6201527714729309, 0.04110288247466087, 0.42763397097587585, 1.6019501686096191, -0.3970277011394501, -1.2884513139724731, -0.4103486239910126, -1.3541442155838013, 0.20465701818466187, -1.121226191520691, 0.7013086080551147, 0.5079914927482605, 1.0504056215286255, 0.41274866461753845, 0.11189696192741394, -1.7117441892623901, 1.6244744062423706, -1.7228035926818848, 0.1486036479473114, 0.3020440638065338, -0.7035570740699768, -0.3226918578147888, -0.33732983469963074, 0.25865864753723145, -1.5051895380020142, -0.41120901703834534, -1.125134825706482, 0.736214280128479, 1.1302950382232666, -1.0765012502670288, -1.8540862798690796, 0.3980903625488281, 0.48694008588790894, 1.2704812288284302, 1.798236608505249, 0.1259712427854538, -0.646588146686554, 0.09050221741199493, -0.322824627161026, -0.7179615497589111, -0.78254234790802, 0.17343968152999878, -0.8128791451454163, 0.7248539328575134, 0.33263877034187317, 0.09078828245401382, 1.0402462482452393, -1.4301851987838745, -1.334895133972168, 0.8227076530456543, 1.9791452884674072, -0.27147915959358215, -0.8909919857978821, 1.8775790929794312, 0.3430365025997162, -0.9132824540138245, -1.1292002201080322, 0.26046180725097656, 0.09857209026813507, -0.5858811140060425, -0.8737637400627136, -1.7182587385177612, 1.068919062614441, 1.0584304332733154, 0.9058506488800049, -0.46065691113471985, 1.1726493835449219, -0.6626402139663696, -0.6059598922729492, 1.3021132946014404, -0.3367089033126831, -1.5757697820663452, 0.3026125431060791, 0.9899929165840149, -0.08750467747449875, -2.333796262741089, 1.2497928142547607, 0.08703631162643433, 0.7698201537132263, -1.7193604707717896, -0.7059505581855774, 0.5547540783882141, -0.4895179569721222, -0.16565145552158356, -0.03676048666238785, 0.5992739200592041, -1.1725354194641113, 0.1958693116903305, -0.7409565448760986, -1.0412877798080444, -0.3735387921333313, 0.16767384111881256, 0.7302685976028442, 1.1264950037002563, 0.3259781301021576, 1.2661736011505127, 2.493008613586426, -0.5642350912094116, 0.3812662959098816, -0.3854677677154541, 1.6021462678909302, -0.8556393980979919, 0.8738681077957153, 0.37690678238868713, -1.0525388717651367, -0.5496441125869751, -0.7824406027793884, -0.6439560055732727, -0.8684948086738586, -0.5364643335342407, 2.3273656368255615, 0.606595516204834, -0.3977199196815491, -0.56801837682724, -1.2331253290176392, 0.6501784324645996, 0.7660275101661682, 0.3531704246997833, 1.148064374923706, 1.30326247215271], [1.7556712627410889, 0.27754709124565125, -0.564618706703186, 0.37296146154403687, 1.0629031658172607, -0.4765092432498932, 1.338579773902893, 2.1297144889831543, -1.8729861974716187, 2.615128517150879, -0.5214205384254456, 0.8277781009674072, -0.48812490701675415, 0.9409146308898926, 0.44103875756263733, -2.0145111083984375, -0.03730561211705208, 1.0395992994308472, 0.7822389602661133, -1.274876356124878, -1.3218605518341064, -1.149839162826538, -0.5918534398078918, 0.12347373366355896, -0.23883481323719025, -0.4799124002456665, -0.5177185535430908, -1.4250781536102295, 0.943020224571228, -0.5224285125732422, -0.5412615537643433, 0.68552565574646, -1.7276661396026611, -0.351447731256485, 0.5136277079582214, -0.04397653043270111, -1.8424019813537598, 1.8161673545837402, -1.3517370223999023, -1.3205394744873047, -0.3340688645839691, 0.01835050992667675, -0.055188123136758804, 0.9402804374694824, -0.07982238382101059, -0.6225053071975708, 0.6791321039199829, 2.0327677726745605, -1.2892236709594727, -1.3067522048950195, 0.9451497197151184, -0.8937757015228271, -0.9747244119644165, 0.4583772122859955, -1.4282232522964478, 0.9198121428489685, -1.3754112720489502, 0.8576692342758179, 1.0229154825210571, 0.6039249897003174, -0.668990969657898, 0.8691812753677368, -0.9728406667709351, -0.17120055854320526, -1.2790050506591797, 0.15332771837711334, 0.9320427775382996, 0.3325519561767578, 0.09431304782629013, -1.3251715898513794, -1.8749717473983765, 1.3616375923156738, 1.2470427751541138, -0.3570865988731384, 0.8416494131088257, -0.550366997718811, 0.34177252650260925, 0.3017260730266571, -0.5436804890632629, 0.5268807411193848, -2.0540387630462646, -0.640071451663971, -0.29859739542007446, 0.2475830465555191, -0.8509341478347778, 0.22222712635993958, -1.6135562658309937, -0.40189826488494873, -0.5076345801353455, 0.12513665854930878, -0.07409712672233582, 0.25710272789001465, -1.1773101091384888, 0.7875933051109314, 0.436920166015625, -2.7946462631225586, 0.9820879697799683, -0.2507959008216858, -0.4788317382335663, -1.44699227809906, 1.4849188327789307, -0.873692512512207, -0.7241269946098328, 1.0247787237167358, 3.1798276901245117, -1.0061415433883667, 0.032393716275691986, -1.3295488357543945, 0.5916672945022583, 0.3653275668621063, -1.2032697200775146, 0.35824504494667053, -0.8503055572509766, 0.9632182121276855, 0.10139956325292587, -0.5247623920440674, -1.43878173828125, -0.5383896231651306, -0.9594379663467407, 0.12438174337148666, 1.124313473701477, -0.19461020827293396, 0.42351287603378296, 1.3025258779525757, 0.9812647700309753, -0.18450263142585754, -0.5226386785507202, 0.06921349465847015], [-0.972160279750824, -0.3500429689884186, 1.1678305864334106, 1.5255000591278076, 1.021814227104187, 1.7842440605163574, 2.214503526687622, -0.11127366870641708, -0.6099086999893188, -1.347455382347107, -1.1611846685409546, 0.5052098631858826, -0.6897922158241272, 1.398290753364563, -0.47502225637435913, 1.8476569652557373, -1.2453923225402832, 0.5295153856277466, 0.041794177144765854, -1.4289723634719849, -1.0689706802368164, -1.1926466226577759, -0.031974878162145615, 0.6320939064025879, -1.0220764875411987, -1.0553805828094482, 0.31448662281036377, 0.7100869417190552, -1.0228767395019531, -0.8712555170059204, -0.12113222479820251, -1.2288758754730225, 1.754612922668457, 0.6528047919273376, -0.014494849368929863, -0.7753233313560486, 1.9297528266906738, 1.023201823234558, 1.0673155784606934, 0.8268429040908813, -1.9419548511505127, -0.297039657831192, -0.07204651832580566, -1.8376260995864868, 0.6955649852752686, 1.3366811275482178, 0.7053126096725464, -1.7145200967788696, -0.6096802353858948, 1.081678032875061, -1.844522476196289, -0.375810831785202, 0.9349539279937744, -0.7757200002670288, -0.5147177577018738, 1.531552791595459, 0.25156089663505554, 0.5176905989646912, 0.10323791205883026, 0.5586642026901245, -0.46934545040130615, 0.22428056597709656, -0.8346437811851501, 1.0047568082809448, 0.033746782690286636, -0.4497108459472656, -0.32016271352767944, 1.1483362913131714, -0.28025639057159424, -0.24758104979991913, -0.3071444034576416, -1.5154739618301392, 1.0897929668426514, -1.1276758909225464, 0.8038402795791626, -0.29263755679130554, 0.0411265604197979, 0.33958953619003296, 1.4407851696014404, -1.4380536079406738, 0.19638463854789734, 0.5613980889320374, 0.5088040828704834, -1.6684623956680298, -1.1062932014465332, 0.25165680050849915, -1.4818934202194214, -1.062313199043274, 1.6158424615859985, -1.6050355434417725, -0.7127230763435364, -2.2025842666625977, 1.5704524517059326, 1.8994609117507935, -0.5501918792724609, -1.578287124633789, 1.0398825407028198, 0.2777967154979706, -1.0408717393875122, 0.9517881870269775, 0.3036648631095886, 0.8248509168624878, 1.30094575881958, 0.19209028780460358, -0.7359718084335327, 1.9728868007659912, -1.2789993286132812, 1.0248576402664185, 1.6040774583816528, 0.6257633566856384, -0.7534840106964111, -0.4581840932369232, 1.7490440607070923, -2.1933155059814453, -0.10519971698522568, 1.4375088214874268, 0.19653227925300598, 2.3463218212127686, -2.1328744888305664, -0.11968038231134415, -1.058786153793335, 0.5861961245536804, -0.10388068854808807, 0.9434962868690491, -0.0524403378367424, 1.9519171714782715, -0.3633963465690613, -0.9916825294494629]], [[0.2900997996330261, 0.19131647050380707, -1.744175672531128, -1.6644470691680908, 0.9306942224502563, -0.459711492061615, -0.5969265103340149, -0.4648151695728302, 1.952834963798523, 0.5984652638435364, 0.21557928621768951, 0.5812435150146484, -0.7254699468612671, -1.4332358837127686, -0.11357622593641281, 1.2030097246170044, 0.29624947905540466, 0.8703567981719971, 0.5039858222007751, 1.5453892946243286, -1.9173327684402466, 1.282596468925476, -1.1557101011276245, -0.09098149836063385, 0.021841635927557945, 0.014815703965723515, 0.5778564810752869, -0.7286205291748047, 1.0243312120437622, -1.014689326286316, -1.5194921493530273, 0.09592137485742569, -0.4812304377555847, -0.46225282549858093, -0.07368078827857971, -0.04309575632214546, -0.019108125939965248, -0.9169886112213135, -2.5976903438568115, -1.3252077102661133, -0.7294141054153442, -1.7705738544464111, -0.6760648488998413, -0.05193435400724411, -0.4535287022590637, 1.4516593217849731, -0.29711446166038513, -0.31324222683906555, 0.11271946877241135, 0.09948384016752243, -0.32602769136428833, -1.2787134647369385, -1.9219670295715332, 0.3742511570453644, -0.989344596862793, -1.2849477529525757, -0.9602864384651184, -0.7419907450675964, -1.152853012084961, -0.907235324382782, -0.45019224286079407, 0.037717055529356, -0.8826646208763123, -0.9413135051727295, 1.1882232427597046, 1.01716947555542, -0.7637490034103394, -0.2931636571884155, -0.045276299118995667, -0.686393141746521, -0.2623271346092224, 1.950594425201416, -0.5458753705024719, -1.31844162940979, -1.588672161102295, 0.6367091536521912, 1.055423378944397, -1.4870259761810303, -0.6582580208778381, -0.28174087405204773, -0.48785465955734253, -0.8117080926895142, 0.6879262328147888, -0.13931842148303986, 1.026090383529663, -0.46917206048965454, -3.679067611694336, -0.0618712455034256, 1.4785819053649902, 0.17962029576301575, 0.4756004512310028, -2.2624588012695312, 0.7984906435012817, -1.2872226238250732, 0.37922143936157227, 0.2572287619113922, -0.47148528695106506, -0.003956148866564035, 0.42988690733909607, -0.4098801016807556, 0.34415188431739807, 0.26966592669487, 0.3000487685203552, 0.2767840623855591, 1.397519826889038, -0.8218969106674194, 0.6658868193626404, 0.09921339899301529, -0.09269390255212784, -0.7627812027931213, -0.70573890209198, -0.47620251774787903, 0.9097482562065125, 0.3260285258293152, 0.6198534965515137, -1.5532439947128296, -1.1376311779022217, 0.7518377304077148, 0.46312880516052246, 0.8186694979667664, -0.6045809984207153, 1.5531195402145386, 0.34230178594589233, -0.6443436145782471, -0.08300129324197769, -1.3761159181594849, 1.376729965209961, 1.1982347965240479], [0.4999066889286041, 0.5340954661369324, 1.380843162536621, -0.5515047907829285, -0.1524970382452011, -0.7688418626785278, -0.7111241817474365, -0.16001243889331818, -0.05955466628074646, 0.48364636301994324, 0.06565890461206436, -1.4587905406951904, 0.13147471845149994, -0.8099920749664307, -1.1152647733688354, 0.6049820184707642, 0.13386327028274536, 0.552574098110199, 0.2070057988166809, -0.6908044219017029, -0.6291459798812866, -0.46171990036964417, -1.5826090574264526, -0.907434344291687, 0.7983424663543701, -0.2078579217195511, 0.14821328222751617, 0.7030150294303894, -1.6953599452972412, -0.7823086380958557, -1.3194667100906372, 1.0838673114776611, 0.3454430103302002, 2.006239891052246, -1.1252050399780273, 0.31036239862442017, -0.420954167842865, -0.9160901308059692, 0.16483759880065918, 0.002691546455025673, -1.1617038249969482, -1.0248591899871826, -0.34003573656082153, -0.2788456380367279, -0.3642967641353607, -0.5800682902336121, 1.1773127317428589, 0.5954756140708923, 0.05630958080291748, 0.9356659054756165, -0.9700098633766174, -0.22011685371398926, -0.627246081829071, 0.010755983181297779, -0.7896903157234192, 1.4141212701797485, -0.4810485243797302, -0.6662404537200928, -1.7527724504470825, 0.25594329833984375, -1.6668463945388794, 1.104408860206604, 1.4654630422592163, -1.1177953481674194, 0.7062392234802246, 0.021130260080099106, -1.1385339498519897, -2.363480567932129, -1.055189609527588, 1.0897434949874878, -1.0709093809127808, -0.25432875752449036, 0.34756380319595337, 0.1635531634092331, 0.0828266367316246, -1.504583477973938, 0.5314857959747314, -1.2175348997116089, -0.2657012939453125, -0.3768883943557739, 0.5808904767036438, -0.19178485870361328, 0.12933847308158875, -0.997796356678009, -1.6418707370758057, -0.454620361328125, -0.9966447353363037, -0.07866879552602768, -0.36556246876716614, 1.0554710626602173, -0.0016404371708631516, 0.4074570834636688, 1.0242739915847778, -1.5967012643814087, 0.9111329913139343, -0.23578478395938873, -0.991489052772522, 0.25036919116973877, -0.10830529779195786, 0.8364410400390625, 0.9179093837738037, 0.21494625508785248, 0.18862153589725494, -1.1657792329788208, -1.2801893949508667, 1.5322437286376953, 1.1590325832366943, 1.1233221292495728, -0.3253464698791504, -0.28238487243652344, 0.05910815671086311, 0.6335570216178894, -0.9036139845848083, -1.2383148670196533, 0.3628619909286499, 1.430683970451355, -1.1550214290618896, -0.7418856620788574, 0.6494945883750916, -0.45002856850624084, 0.30333176255226135, 0.7568125128746033, 0.6641928553581238, -2.610621452331543, 0.3174094557762146, 0.4087132513523102, 0.8342999219894409, 0.1526113599538803], [-0.48925912380218506, 0.9609582424163818, 0.1312279850244522, -1.5423389673233032, 1.0247361660003662, -0.2566244900226593, -1.5310564041137695, 1.5256181955337524, -1.4615981578826904, -0.8564333319664001, -0.3707873821258545, -0.12323644757270813, 0.7056399583816528, -0.41040298342704773, 1.4303548336029053, 0.11987485736608505, -0.9550960063934326, -0.15348148345947266, 1.380454421043396, -0.2671393156051636, 0.08435545116662979, -1.5248796939849854, 1.1982673406600952, -0.33738330006599426, -1.3933113813400269, 0.6118906736373901, -0.05156252533197403, 0.9112042188644409, -0.6932428479194641, -0.5575324296951294, -0.2940944731235504, -0.006976124830543995, -0.8622162342071533, -0.11388633400201797, -1.0774387121200562, 1.7727352380752563, -1.1729236841201782, -0.8477965593338013, 0.27140969038009644, 0.9610301852226257, 0.6351974010467529, -0.5737982988357544, 1.41275155544281, 0.5872961282730103, 2.059082269668579, 1.4870648384094238, 0.44835612177848816, -2.8328800201416016, -0.24249625205993652, -0.9570533633232117, 0.621071994304657, -0.15235885977745056, 1.381799578666687, -0.09207219630479813, -0.9038788080215454, -0.9594529271125793, 3.061572551727295, 0.03662882000207901, -1.1750868558883667, 1.187714695930481, 0.19380803406238556, -0.008611306548118591, -2.5464065074920654, -0.34344181418418884, -1.438567876815796, 0.5365654230117798, -2.3207480907440186, -0.6451183557510376, -0.45769786834716797, -0.07501204311847687, 0.8664109110832214, -0.8536257743835449, 0.2890034019947052, 1.323438286781311, -0.6627035140991211, 2.285874128341675, -1.0431792736053467, 0.6008399128913879, -1.0820581912994385, -1.9972950220108032, -0.9144842624664307, -0.4797173738479614, -0.19524872303009033, 1.600008249282837, -0.6887297034263611, -1.0191280841827393, 1.7461967468261719, -1.0016013383865356, 1.4483493566513062, 0.7231431007385254, -0.0731273964047432, 0.04408338665962219, -0.4511262774467468, -1.5606368780136108, 1.2100151777267456, 0.5379788875579834, -1.7628980875015259, 0.44048577547073364, -0.9001640677452087, 0.37817883491516113, -0.5755505561828613, 0.870158314704895, -0.6603551506996155, -0.16127757728099823, -1.6941595077514648, -0.5758056044578552, 0.4140796959400177, -0.2654252350330353, -0.5217928886413574, -0.5803544521331787, 0.39547380805015564, 0.2956898510456085, 0.7914918661117554, -1.479466438293457, 0.5878609418869019, 0.5770025849342346, -0.8191671371459961, -0.11390262097120285, -1.2435287237167358, -1.7914316654205322, -1.0203020572662354, 0.7134718894958496, 0.942359983921051, 0.1745205968618393, 1.3361660242080688, 0.05652816966176033, -0.8263663053512573, 0.17807736992835999], [-1.1835073232650757, 0.7121251225471497, 0.44221383333206177, -1.6745882034301758, -2.599170684814453, 0.6139664649963379, 0.5547922849655151, 0.9561732411384583, 1.1346465349197388, -1.5547044277191162, 0.013513066805899143, -0.4133281111717224, 1.0621531009674072, -0.7798900604248047, -0.9213404655456543, 1.3508460521697998, 1.764334797859192, -0.3976220190525055, -1.156226634979248, 0.32617974281311035, -1.181880235671997, -1.3093982934951782, -0.5177274346351624, -2.4021739959716797, -0.877681314945221, 2.589202404022217, -0.6640554666519165, -0.0561661534011364, 0.9360652565956116, 0.5328702330589294, -0.4307299256324768, 0.539053201675415, -1.2014224529266357, -1.766678810119629, 1.0543479919433594, 2.5025596618652344, 0.7436617016792297, -0.1452857255935669, 0.8754297494888306, -1.0158673524856567, 0.7199941277503967, -0.06942673027515411, 1.3691654205322266, -0.7201223373413086, -0.46419668197631836, -0.5268737077713013, 1.3552474975585938, 0.9994326233863831, -0.45690903067588806, -1.2325634956359863, -0.08402916043996811, 1.7066988945007324, -0.8147334456443787, -1.5248104333877563, 0.05268010497093201, -1.1935089826583862, -1.3138707876205444, 0.5719462037086487, -0.48558634519577026, -0.6415722966194153, 1.1946978569030762, 0.3782348036766052, -1.5966516733169556, 0.7637073397636414, 0.8130876421928406, 0.3274577260017395, 0.19178877770900726, -0.36600056290626526, 2.0459585189819336, 0.45933204889297485, 0.16268280148506165, -0.060990095138549805, -2.1815547943115234, -1.4291530847549438, 0.78989177942276, 0.5232966542243958, -1.078235387802124, 0.14626047015190125, 0.6144775152206421, 0.27357637882232666, -1.0151309967041016, 1.8954743146896362, -1.961732268333435, 0.9738766551017761, -1.068927526473999, 0.09527113288640976, 0.028622804209589958, 0.057453423738479614, 1.3916996717453003, -0.7890096306800842, -0.4807688593864441, -0.4633495807647705, 0.6155312061309814, 0.880549430847168, 1.3953449726104736, 0.5234401822090149, 2.529512643814087, -0.47005802392959595, 1.2855725288391113, -0.9511357545852661, -0.45628511905670166, 0.9780234694480896, 2.2433669567108154, -0.33796370029449463, 0.5040194988250732, -1.648708462715149, -0.6757015585899353, 1.6213527917861938, 0.45123687386512756, 0.518153965473175, 0.23886016011238098, 0.05351715534925461, 0.7681190967559814, 0.08314895629882812, 0.681640088558197, -1.17740797996521, 3.0192015171051025, 0.07067758589982986, 0.16166500747203827, 0.7620725035667419, 0.7091426849365234, -1.6858831644058228, -0.07550132274627686, 0.9938201904296875, -0.8777374029159546, -1.4049829244613647, 0.714657723903656, 0.33172109723091125], [1.0324293375015259, -1.1614550352096558, -0.215937539935112, 1.27390456199646, -0.4858584403991699, 0.8030405044555664, 0.6755856275558472, -0.8472784757614136, 0.8162170052528381, -0.22944101691246033, -0.1762549877166748, 1.4238373041152954, -0.4937637746334076, 1.6330392360687256, -1.0780308246612549, -0.9933891296386719, 0.3554883897304535, -0.14587010443210602, 1.958661437034607, 0.0443294495344162, -1.1566007137298584, -1.847056269645691, 0.7209595441818237, -0.6346396803855896, 2.0950253009796143, 0.15056119859218597, -0.12640812993049622, -0.5943643450737, 0.41006147861480713, -0.9876232743263245, -0.41666316986083984, -0.12047957628965378, -1.44920814037323, 1.697641134262085, -0.7583232522010803, 0.9059156775474548, -0.2561038136482239, 0.33088386058807373, 1.1391609907150269, -0.25124454498291016, -1.0057564973831177, 1.2756599187850952, -1.7679266929626465, -0.7447313070297241, 0.9057657122612, -1.7236407995224, 0.5509727597236633, 0.2696024477481842, -0.20378638803958893, 0.16777531802654266, 1.6586662530899048, -0.9579013586044312, -0.0305153988301754, -0.677994430065155, -0.1894683986902237, 0.23738005757331848, 0.3112664818763733, 0.34099528193473816, -0.9705223441123962, 0.5726091861724854, 1.365388035774231, -0.4709508717060089, 1.5959317684173584, 0.9256798624992371, -0.04803925380110741, -0.9729027152061462, -0.49905091524124146, -0.7377269268035889, 0.42962753772735596, -1.0250338315963745, -1.2637908458709717, 0.550411581993103, 0.6873824000358582, -1.269034743309021, 1.6445080041885376, 2.5951383113861084, 1.5248128175735474, -0.07010867446660995, -0.42355772852897644, 0.5087162852287292, -1.1244704723358154, -1.6986815929412842, -0.4247811436653137, 1.3401395082473755, -0.6380009055137634, 1.0081440210342407, 0.9662337303161621, 0.13034625351428986, -0.5124287605285645, 2.6010255813598633, 1.1702030897140503, 0.036915332078933716, 2.2415523529052734, 1.176393985748291, -0.5231649279594421, -0.8306981921195984, 0.09253710508346558, -0.7395046353340149, 0.6139577627182007, 0.9134973287582397, -1.4333076477050781, 0.29823264479637146, -0.08447592705488205, 0.42992791533470154, -0.6346613764762878, 1.4704116582870483, -1.148890733718872, 0.3647836446762085, 0.2172461301088333, 1.018855333328247, 1.9311256408691406, 0.4941296875476837, -0.7047876715660095, 1.17507004737854, 1.7236741781234741, -0.7484036087989807, 0.45306745171546936, -1.1826492547988892, 0.7043898701667786, -0.9514672160148621, 0.7416298389434814, 1.0309070348739624, 0.31860437989234924, 0.7968588471412659, -0.39031854271888733, -0.7885724902153015, -1.2804677486419678, -1.0430222749710083], [-0.9120847582817078, 1.450722336769104, -2.3593411445617676, 1.1796975135803223, 0.08050097525119781, 1.567175269126892, -1.135080337524414, 1.0651671886444092, -0.8644511103630066, 0.41893449425697327, -0.29788827896118164, 1.1290581226348877, 0.9615883827209473, -0.4737183451652527, 0.43582162261009216, -0.03733493387699127, -0.30540093779563904, -1.7621917724609375, -0.3181056082248688, -0.02595771662890911, -2.4313933849334717, 0.8888700604438782, 1.1231104135513306, 1.0687508583068848, -0.23578347265720367, 1.7075140476226807, 1.006471037864685, -0.49691882729530334, 1.5147336721420288, 0.032140593975782394, -0.6725347638130188, -0.7337876558303833, 1.8828513622283936, 0.23144713044166565, 1.662001371383667, -1.1795364618301392, -1.3636767864227295, -1.2202173471450806, -2.4003870487213135, -0.9248567223548889, -0.7210448980331421, 0.5230192542076111, -1.1548174619674683, 0.028790831565856934, -0.8907812833786011, 1.5438780784606934, -1.1850144863128662, 0.9057270288467407, 2.3955883979797363, -1.3783282041549683, -1.3103216886520386, 0.6613067388534546, 0.2535344660282135, -0.320583313703537, -0.1878560185432434, 0.07986826449632645, -1.0048305988311768, 1.6646909713745117, 0.2032678872346878, -0.5395162105560303, 0.12219356000423431, 1.3252345323562622, -0.5224721431732178, 1.3979521989822388, -0.7747567892074585, -0.6817591190338135, -1.572481632232666, -0.13461020588874817, -1.1515321731567383, 0.9171082377433777, -0.7491622567176819, 2.8520874977111816, -0.7932415008544922, 1.741987705230713, 0.368911474943161, -0.8467163443565369, 0.8284722566604614, 1.9939478635787964, -1.129986047744751, 0.5416487455368042, 0.11778150498867035, -0.30942878127098083, 0.3851335644721985, -1.3111850023269653, 0.6479909420013428, 1.4145550727844238, 0.5982085466384888, 0.34297534823417664, 0.9900728464126587, 0.45098158717155457, -0.08139988780021667, 0.6518352627754211, 0.9609126448631287, 0.07154150307178497, -0.11605044454336166, 0.15353035926818848, -0.3654860556125641, -0.8988369703292847, -1.1756930351257324, 1.529963731765747, -0.822324812412262, -0.543924868106842, -0.03918290510773659, 0.01137456949800253, -0.2638038396835327, -0.0540769137442112, 0.7434355020523071, -0.7375478148460388, -1.3920496702194214, 0.27513036131858826, 1.7271381616592407, -0.03703460097312927, 0.4638817012310028, 1.497531771659851, -0.20880018174648285, -1.016237497329712, 0.966060221195221, 1.384908676147461, 0.864450991153717, 0.7278337478637695, -0.5447567701339722, -1.1040151119232178, -0.3962826430797577, -2.7029850482940674, -0.6120663285255432, 0.8105989694595337, 1.2188210487365723, 1.134225606918335], [-0.8689504265785217, -0.2746170461177826, -0.8223097324371338, -0.39464297890663147, 1.0478838682174683, -0.36201411485671997, -0.037818294018507004, 0.4963311553001404, -2.574706554412842, -0.3516154885292053, -1.8593114614486694, -0.19420720636844635, 0.48298609256744385, 0.5311814546585083, 0.2640577256679535, 0.09347958862781525, -0.1740589290857315, -1.7807908058166504, -0.5296299457550049, -1.4487229585647583, 1.030094027519226, -0.15302374958992004, -0.536535918712616, 0.971869170665741, 1.540683388710022, -0.45310673117637634, 0.1565098911523819, -2.5114786624908447, 1.4912612438201904, 0.12223386764526367, -0.24481013417243958, -0.8415794372558594, 0.2529292404651642, 0.16218450665473938, -0.34066417813301086, 0.5673283934593201, 1.3385289907455444, -1.0877947807312012, -0.6869934797286987, -1.067476749420166, 0.566511332988739, 1.1717594861984253, -1.02992844581604, 0.7665131092071533, -0.562671422958374, 0.9118315577507019, -1.3567880392074585, 0.902833878993988, -0.14582635462284088, -1.2717304229736328, 0.04840066283941269, -0.46458908915519714, -1.7222142219543457, -0.67974853515625, -0.1262965053319931, 2.502593994140625, 0.40631213784217834, -0.4554709494113922, -0.2054283171892166, 0.999718427658081, -1.6290514469146729, 1.0141420364379883, -0.31973543763160706, 0.0420956127345562, -1.5856279134750366, -0.2804751992225647, -1.1253443956375122, 0.4396101236343384, -0.8757112622261047, -0.04230429232120514, 0.2875065207481384, -0.9036579728126526, 2.4758105278015137, -0.8208642601966858, 0.6377661228179932, -0.7842898368835449, 0.005853333510458469, 2.5525734424591064, 0.6572259068489075, 0.9653492569923401, 0.061435043811798096, -0.24059396982192993, -1.543210744857788, 0.711487889289856, 0.7706249952316284, -1.0533405542373657, -0.8451381921768188, 0.5910260677337646, 2.1634225845336914, -0.9612704515457153, 1.0100560188293457, 1.0666038990020752, 0.6340692639350891, -0.11551915109157562, 0.47529125213623047, 0.3851662278175354, 0.10310228168964386, -0.44654905796051025, 0.7961313128471375, 0.7988362312316895, 1.262214183807373, 0.13944892585277557, -0.4562996029853821, -1.052384614944458, -1.0857166051864624, 0.5890406370162964, 0.8332500457763672, 0.7478470206260681, -1.1786301136016846, 1.544374942779541, -1.4662889242172241, 0.3587053716182709, -0.1006108894944191, -1.7641311883926392, -0.7037970423698425, 1.3475204706192017, -0.7358949780464172, 0.3660666346549988, -1.4863072633743286, -0.6329774856567383, 0.5960929989814758, 0.6229690909385681, 0.6846696138381958, -2.3608312606811523, 0.14679737389087677, -0.03405499830842018, 1.1740927696228027, 0.3114299774169922], [-0.9477525353431702, 0.16052068769931793, 0.16339729726314545, -0.8351878523826599, -2.7690231800079346, -0.5754214525222778, 0.3571224510669708, 1.690970778465271, 0.5610281825065613, 0.10190989077091217, 0.250161349773407, 0.5195735096931458, -1.483298420906067, -0.9443438649177551, -0.8785057663917542, -0.8728160858154297, -0.8048092126846313, -1.0675143003463745, -2.4250190258026123, -0.16647489368915558, 0.0971095934510231, -1.1949422359466553, 0.45455819368362427, -0.053855717182159424, -0.1285388171672821, 0.9659866690635681, 0.028577201068401337, -0.9012709856033325, -0.09970220923423767, 0.38537490367889404, -0.14368808269500732, 0.152807354927063, -0.18708068132400513, -1.3645687103271484, -0.7250715494155884, -0.3247777223587036, 1.3870331048965454, 0.9973262548446655, 0.030234968289732933, 0.9093000292778015, 0.7084673643112183, -0.28600284457206726, 0.3395053744316101, -0.9227783679962158, -1.451178789138794, -0.22092880308628082, -1.3877514600753784, 1.3709702491760254, -1.2208521366119385, -1.2075419425964355, -0.851950466632843, -0.9843329787254333, 1.5502800941467285, 0.10250777006149292, 0.8901785612106323, 0.24919749796390533, 1.1038670539855957, 0.507330596446991, -0.9868624806404114, -0.1642046719789505, 0.05485948547720909, -0.7366952300071716, -2.254603385925293, 0.7892338633537292, 1.4444390535354614, -0.41406458616256714, 3.358372926712036, -0.010319527238607407, 0.07402769476175308, 0.473279744386673, -0.9137392640113831, -0.8939041495323181, -1.2978007793426514, -0.024154623970389366, 0.24950739741325378, -1.3512030839920044, -1.1046860218048096, -0.13942454755306244, 0.8957763910293579, -0.4018552899360657, 0.048725757747888565, -0.2427828162908554, 1.0347472429275513, 1.204028606414795, 0.4489041864871979, 0.08888986706733704, -0.3542596399784088, -0.8474798798561096, -0.10219905525445938, -1.1551241874694824, -0.3414446711540222, 0.08130205422639847, -1.3192466497421265, -1.132482886314392, -1.2081950902938843, -0.034047603607177734, -0.4322066009044647, 1.7098275423049927, -0.6920502781867981, -0.739663302898407, -0.4184371829032898, 0.4642229378223419, 1.9240093231201172, 0.457051545381546, 1.2303466796875, -1.4416910409927368, -0.3152574300765991, -0.9880529642105103, -0.3340420722961426, 1.5266776084899902, -0.6179490089416504, 0.0023770141415297985, 0.9314723610877991, 1.048365831375122, 0.17482890188694, 1.5295766592025757, 0.9037837386131287, 1.0840035676956177, 0.12487722933292389, -0.1454494148492813, -1.2848621606826782, -1.1690187454223633, -0.1785375475883484, -0.37575629353523254, -0.24955151975154877, -0.2917926609516144, -1.8658263683319092, -0.8685834407806396]]], [[[-1.162335753440857, -0.7625601291656494, -0.44518420100212097, 0.25788745284080505, 0.9785229563713074, 0.10711058974266052, 0.5788000226020813, -1.724564790725708, -1.6180580854415894, -0.24970974028110504, -0.7962983846664429, -0.3039931058883667, 0.9713898301124573, -0.03990057855844498, 0.45520272850990295, 1.1545788049697876, -0.28996071219444275, 3.2143568992614746, -0.030070826411247253, -0.3350050151348114, -0.16119961440563202, 1.3276057243347168, 0.18545715510845184, -0.6780622005462646, 0.23199345171451569, 1.410524606704712, 0.4828125536441803, 0.6139604449272156, -0.4626713991165161, 0.18509997427463531, 0.39075765013694763, 0.10683474689722061, 0.22152239084243774, 1.4129022359848022, 0.22181011736392975, 1.404748797416687, 2.3492484092712402, 0.6647399663925171, 0.5819495320320129, 0.8535281419754028, -0.3707146644592285, -0.3021426200866699, 0.4872971177101135, -1.2899370193481445, -0.09438809752464294, 0.07924479991197586, -2.0367400646209717, 0.9885523319244385, 0.39479726552963257, -0.005380451213568449, 1.8257728815078735, -0.7865478992462158, -0.32081958651542664, 0.5216240286827087, -1.6808909177780151, -0.9816949963569641, -0.4366995692253113, 0.2603510320186615, -1.0680897235870361, -0.555016815662384, -0.2832033932209015, 1.494203805923462, 0.24532032012939453, 1.1918554306030273, -1.1243610382080078, -1.0421772003173828, 0.46032217144966125, -0.856275200843811, 1.3964152336120605, 0.5055970549583435, 0.14016889035701752, -0.7986996173858643, -1.868044376373291, -0.27816879749298096, 1.1656606197357178, -2.985987901687622, -1.3252867460250854, 0.18989995121955872, -0.2868749797344208, 0.6037196516990662, 0.11768889427185059, -0.19198383390903473, 0.9557780623435974, -0.6255386471748352, 0.7548840045928955, 1.4503145217895508, 0.4448806047439575, -0.0747564509510994, 1.0137832164764404, 1.6448256969451904, -1.9277254343032837, -0.6272504925727844, -0.07047280669212341, 0.9991206526756287, -1.1661810874938965, 0.4170234501361847, -0.45136967301368713, -1.09222412109375, -1.2888851165771484, 1.0990017652511597, -0.25183501839637756, -0.04266238212585449, 1.0948939323425293, 0.014871313236653805, 0.28055503964424133, 2.3037211894989014, 0.5135699510574341, -0.3846454918384552, -0.0635175108909607, -0.12389431893825531, 0.18652819097042084, 0.5546464323997498, 0.36188003420829773, 0.8386794328689575, -0.7090847492218018, -0.4790046513080597, 1.1447653770446777, -0.9373288750648499, -1.4875015020370483, 0.23254427313804626, -1.1507314443588257, -0.2553330659866333, -0.09053530544042587, -0.12511397898197174, 3.1813268661499023, -0.13227273523807526, 1.7314987182617188, 0.9113458395004272], [-0.5792399644851685, 0.43982023000717163, -1.4256335496902466, -0.7858002781867981, -0.32457542419433594, 0.09646844118833542, 1.1614097356796265, -0.04663202539086342, 1.5132331848144531, -0.04490461200475693, 1.150935173034668, 1.8563956022262573, -1.0155471563339233, -0.018881244584918022, -0.75264573097229, 0.03716159611940384, 0.06881238520145416, 0.6076029539108276, 0.7951083183288574, -1.3598076105117798, 0.3541429340839386, 2.1835336685180664, 1.1155062913894653, -1.5711952447891235, -0.8050141930580139, -1.0409555435180664, 2.5548787117004395, 0.9053448438644409, 0.807508647441864, -1.3743561506271362, -0.3730923533439636, 1.3569868803024292, 0.09476380795240402, 0.07898437231779099, 1.6841284036636353, 0.9680589437484741, -0.6934999227523804, 0.4847089648246765, 1.0054680109024048, 0.2416181117296219, 1.214666485786438, -0.282426655292511, -0.9617522358894348, -0.8871394991874695, -1.00406014919281, -1.3011902570724487, -1.9215412139892578, 1.6263822317123413, -0.16086134314537048, 0.06331454962491989, 0.955473005771637, -0.7998717427253723, 0.567406177520752, 0.21564117074012756, 0.5087541937828064, -0.8039599061012268, -0.6122598648071289, 0.6836554408073425, 0.8298212289810181, -0.2937726378440857, 0.00567359896376729, -0.1023927628993988, -0.3833211064338684, 0.6473978161811829, 0.05284890905022621, -0.5466273427009583, 0.10586980730295181, 1.731920838356018, 0.5214649438858032, -2.9537222385406494, -1.207558274269104, -0.3987632691860199, 0.25913727283477783, -0.8907320499420166, -1.4333168268203735, -0.11487632244825363, -0.7354603409767151, -0.1291790008544922, -1.1746361255645752, -0.2445596605539322, -1.3714070320129395, 1.2866419553756714, 0.41908523440361023, 0.44459161162376404, -0.738342821598053, -0.7303106784820557, 0.027694346383213997, 0.03693532571196556, -1.2670478820800781, -0.39250290393829346, 0.6036280393600464, 0.21749351918697357, 0.0907597467303276, -0.5382177829742432, -0.667370617389679, 0.7720798254013062, -0.6706127524375916, 1.0761830806732178, -0.36608925461769104, -0.342298299074173, -1.8796573877334595, 0.7366774082183838, 0.7417735457420349, 1.4533220529556274, 1.2575312852859497, 0.9594399333000183, 0.5831106305122375, -0.794193685054779, -1.0828831195831299, 0.11228562891483307, -1.279157042503357, -0.3064638674259186, -0.6182326078414917, 1.4090440273284912, 0.620854377746582, -1.4186381101608276, -0.7992009520530701, -0.5556820631027222, -2.752056121826172, 0.9242441058158875, 0.6463472247123718, 0.059742316603660583, -0.34196794033050537, -0.3569270670413971, -1.056551456451416, -1.0188928842544556, -0.15022428333759308, -0.3282089829444885], [-0.1357017606496811, -0.5652728080749512, -0.8482221364974976, 1.8101704120635986, 2.266057252883911, -0.27276545763015747, 0.49535849690437317, 0.4682067036628723, 0.3041183054447174, 0.24790458381175995, 1.3970040082931519, 1.8186137676239014, -1.3295063972473145, -0.06628106534481049, 1.8464821577072144, -1.950987696647644, -0.4156361520290375, -1.5579642057418823, -0.2828327715396881, -0.48094016313552856, -0.5020177364349365, 1.150139331817627, -0.5098968148231506, -0.7158947587013245, -0.7858006954193115, -0.3151891529560089, -0.17494919896125793, 1.0812879800796509, -1.2135553359985352, -0.006558041088283062, -1.2664713859558105, 0.09688909351825714, -0.06925825774669647, 0.9868759512901306, -0.36445051431655884, 0.8829445242881775, 0.38553136587142944, -0.25963419675827026, 1.320119023323059, 1.490410566329956, -1.3495457172393799, 1.4396864175796509, 1.2066882848739624, 0.4962984621524811, 0.35800468921661377, 1.2031713724136353, -0.2839956283569336, 0.2842823565006256, -0.8657957315444946, -1.0956428050994873, -1.8178000450134277, -0.7042976021766663, 0.4641708731651306, -1.210988998413086, 0.7969451546669006, -1.343341588973999, 0.4793429374694824, 1.6412967443466187, -0.44640567898750305, -0.5252412557601929, 0.38898125290870667, 1.6671276092529297, 1.3463822603225708, -0.07211197912693024, 0.6685739755630493, 0.11896375566720963, 0.3192330598831177, 0.0822453498840332, 1.1362674236297607, 0.0922638550400734, -0.5838987827301025, 1.5490694046020508, 0.4161888659000397, -0.9904984831809998, -1.0715316534042358, -1.7898449897766113, -0.14464986324310303, -2.2416253089904785, 0.7652804255485535, -0.11865539103746414, 1.1550458669662476, 0.77786785364151, 0.04188256338238716, -0.6095923781394958, 0.244369775056839, -0.489492267370224, 0.7912688851356506, 0.9349055886268616, 0.8471915125846863, -1.5851885080337524, 1.070434808731079, -0.08335857838392258, -0.011849197559058666, -0.6943589448928833, 0.018616287037730217, 0.10625966638326645, -0.6512053608894348, 0.038972239941358566, -0.1741388589143753, -1.3397672176361084, -0.10908111929893494, 0.8918304443359375, 0.6777260303497314, -0.15376998484134674, -1.361059546470642, -0.62491375207901, -0.36843690276145935, -0.8234022259712219, 0.27835017442703247, 0.7151347398757935, -0.2915152907371521, -0.5052916407585144, 1.2744908332824707, 0.15815915167331696, 0.9348865747451782, 0.17015717923641205, 0.10545774549245834, -0.3075115978717804, -2.2390029430389404, 0.2596120834350586, 2.380103349685669, 0.9927945733070374, 0.8750854134559631, 0.0885147824883461, 0.6678913831710815, -0.2370375096797943, -0.21929779648780823, -0.5645646452903748], [0.20757079124450684, -0.33016204833984375, 0.7553583383560181, -0.84609055519104, 1.2950760126113892, -0.29151350259780884, -0.8362318277359009, 0.3527323603630066, -1.2728334665298462, -0.24908329546451569, -0.2134721875190735, -1.9176255464553833, 0.11937406659126282, 1.4488717317581177, 0.9845037460327148, -2.562659740447998, -0.16995343565940857, -0.8725018501281738, 0.28490519523620605, 0.5986062288284302, -1.5584018230438232, -0.8098360896110535, 0.08711043745279312, -0.21413514018058777, 0.42507749795913696, -1.1372368335723877, 0.80157870054245, 1.0470678806304932, 1.2269748449325562, 1.578848958015442, -0.7852640748023987, 1.37481689453125, -0.150240957736969, 1.0675525665283203, -0.7763071060180664, 0.007970381528139114, -1.8075400590896606, 1.2443870306015015, 0.014415325596928596, 0.8731387257575989, -2.3516933917999268, -0.0964096412062645, 1.4110774993896484, -1.0859156847000122, 0.6947317123413086, 0.5144101977348328, -2.056299924850464, 1.3422768115997314, -1.0674848556518555, 0.4270056188106537, -3.355403423309326, -1.0860941410064697, -0.19619520008563995, 1.9082818031311035, 0.05470266938209534, -1.1523618698120117, -1.9290333986282349, 1.0855120420455933, -2.312190055847168, 1.5590318441390991, 0.8182725310325623, 1.612151026725769, -0.6964203715324402, 1.2295184135437012, -0.30915746092796326, -0.3462393879890442, 0.7949697971343994, 0.5900160074234009, 0.1394551545381546, -0.9446667432785034, 0.8502896428108215, 0.15370479226112366, -0.35417860746383667, -0.16648517549037933, -1.5975890159606934, 0.674075186252594, 0.6176983714103699, 0.04361710697412491, -2.263444185256958, -0.279395192861557, 0.21813592314720154, -0.4793519973754883, -1.4537792205810547, 0.23625019192695618, -2.411301851272583, -1.1723452806472778, -2.291177749633789, -0.20821942389011383, -0.04365839809179306, -0.42676693201065063, -0.2996987998485565, -0.0856749564409256, -1.5189666748046875, -0.39479824900627136, -1.114288568496704, 0.46280437707901, 0.08316896110773087, -0.18794800341129303, 1.211437702178955, 0.07255379855632782, -1.1924614906311035, 0.9424579739570618, 0.3922305703163147, 0.5598172545433044, -1.129465937614441, 1.220055103302002, 1.2563426494598389, -0.7164256572723389, 1.1270564794540405, 0.2984926998615265, 0.33783861994743347, 1.6308727264404297, -0.42062267661094666, 0.08488394320011139, -0.6553306579589844, -0.6896849274635315, 0.8094308376312256, -0.8016381859779358, -0.6607874035835266, -0.4793117046356201, 0.24241171777248383, -0.5724248886108398, -0.10470080375671387, 0.5350000262260437, -0.7917149662971497, -1.7895135879516602, 1.1016669273376465, -0.879416823387146], [0.35521745681762695, -1.4966952800750732, -0.5352216362953186, -1.4682589769363403, 1.6668131351470947, 0.7479786276817322, -0.08512585610151291, -1.398454189300537, -0.4354895353317261, -0.3832029700279236, 1.97464919090271, 0.5864238739013672, 0.034312207251787186, -1.585028886795044, 1.2891381978988647, -1.1480897665023804, -0.5263285636901855, -0.5970978140830994, 1.323735237121582, 0.2435319870710373, 0.049202121794223785, 1.774240493774414, -0.48574888706207275, 0.7632221579551697, -1.8949813842773438, 0.10268139839172363, 0.5777002573013306, -0.43965640664100647, 0.6779307126998901, 0.05189875513315201, -1.9278697967529297, -0.13699227571487427, 1.5205196142196655, 0.3145207464694977, 0.7113097906112671, 0.45564451813697815, 0.13611574470996857, 1.3901357650756836, -0.6652541160583496, -0.10038357228040695, 0.7188175916671753, 0.10367389768362045, -0.33932003378868103, 0.8039201498031616, -0.5181899666786194, 0.7199712991714478, 1.9763810634613037, 0.7963135838508606, -0.771904468536377, -0.6500137448310852, -0.5442289113998413, -1.459370493888855, -0.4944173991680145, 1.354427456855774, -0.08221409469842911, 1.731016993522644, 0.19597993791103363, -0.28447744250297546, -0.24513131380081177, 0.6278811693191528, 1.4245280027389526, -1.4089118242263794, -0.6997709274291992, -0.3583971858024597, 1.332289218902588, -0.455025315284729, 0.8027158379554749, -1.2020974159240723, -0.21227127313613892, 0.5231271982192993, 1.4522775411605835, -1.0336008071899414, 0.012360408902168274, -0.5300331115722656, -1.908075213432312, 1.5843678712844849, -0.1965618133544922, -0.990613579750061, 0.4684463441371918, -0.04410378634929657, 1.7272216081619263, 0.30934959650039673, -1.5661484003067017, 1.06610906124115, 1.4955955743789673, 0.1997898370027542, -0.9010801315307617, -0.22164420783519745, -0.5324512124061584, -1.8918076753616333, -0.7371467351913452, -0.5965982675552368, 0.3372596204280853, 0.030547672882676125, 1.7811528444290161, 0.5229504108428955, -0.5510923266410828, 2.4587762355804443, 0.24328286945819855, 1.1124955415725708, -0.7276006937026978, -0.6435457468032837, -0.2751210629940033, 0.2951093316078186, 1.0438063144683838, 1.8604786396026611, 1.0902831554412842, -0.03750206530094147, 0.5640076994895935, 0.6364309191703796, 0.027420828118920326, 0.06664639711380005, -1.16402006149292, 1.5751370191574097, 0.04493650794029236, 0.5298976302146912, -0.6368540525436401, 0.8120726346969604, 1.6511454582214355, -0.8653626441955566, -0.19543853402137756, 1.4659897089004517, 1.389534831047058, 1.6729822158813477, 0.4155603051185608, 0.4718964397907257, -0.1329742968082428, -0.8922855257987976], [-0.6664524674415588, 0.14382310211658478, 1.0922621488571167, -1.0858148336410522, -0.9097829461097717, 0.20643006265163422, 1.7049721479415894, -0.5351647138595581, 0.6902337074279785, -0.18944090604782104, 0.5929993391036987, -1.0979586839675903, 1.0753334760665894, 0.1224236860871315, -0.5541960000991821, -0.26819920539855957, -1.1074737310409546, -1.9526926279067993, 0.4156789183616638, 0.01829134300351143, 0.490193247795105, -1.170763611793518, -1.3804742097854614, -0.6375630497932434, 0.45379146933555603, -0.28380250930786133, 1.0622382164001465, 0.13405156135559082, 0.6764514446258545, 0.7301057577133179, -1.4639294147491455, 0.08816496282815933, -0.9300758838653564, 1.6215183734893799, -0.058141350746154785, -0.5207117199897766, 1.0154141187667847, -1.4411685466766357, 0.6394761800765991, 0.8048731088638306, -0.45397400856018066, 0.7436725497245789, 1.5232452154159546, -1.1825056076049805, -1.2843186855316162, 0.12504538893699646, -1.126607060432434, 1.0801736116409302, -0.5711143016815186, -0.5643048286437988, 0.7359011769294739, -1.0798057317733765, -0.5150655508041382, 1.5449373722076416, -0.5261061787605286, 0.6108396053314209, 0.25358277559280396, -1.2655783891677856, -0.2584128677845001, 0.12194730341434479, 0.9883841276168823, -1.006333351135254, -1.756556749343872, -0.22077856957912445, 0.22426898777484894, 0.8869081735610962, 0.5961045622825623, 0.8950862884521484, -0.2275511771440506, -0.00047387901577167213, -0.23600563406944275, 2.802821159362793, -0.4176822602748871, -0.35389792919158936, 0.15143722295761108, -0.3105001449584961, -0.5645835995674133, 0.29477939009666443, 0.2313946932554245, -0.3888164162635803, -0.1612681895494461, 0.7178561687469482, -2.193991184234619, -0.6827124953269958, -1.7300045490264893, 0.4196132719516754, -0.8247127532958984, -0.26866206526756287, 0.5026695132255554, 0.9042040705680847, -1.1928685903549194, 0.6243336796760559, 0.4015599489212036, -0.9496049880981445, 0.5754584074020386, -0.41600748896598816, -0.3401925265789032, -0.6253591179847717, -1.626138687133789, -0.4475739896297455, 1.04017174243927, -0.9989467859268188, -0.5627105236053467, -0.7405438423156738, 1.395790457725525, 0.8170791268348694, -0.13098302483558655, 0.13843677937984467, 1.4943535327911377, 0.09257703274488449, -1.0414717197418213, -1.4677715301513672, -0.1618949919939041, -2.2283530235290527, -0.6568993330001831, -0.6729313135147095, 1.6614336967468262, 1.1887742280960083, 1.6256799697875977, 0.2567744851112366, -0.3288753628730774, -0.09998679161071777, -0.8221825957298279, -1.8304611444473267, 1.6660785675048828, -0.5330352187156677, -1.8523424863815308, 1.4457157850265503], [1.9054968357086182, 0.12413979321718216, 0.1955835521221161, -1.428760051727295, -0.8333362936973572, -0.1318548023700714, -0.45660555362701416, -1.5148491859436035, -0.2065601497888565, 1.271804928779602, -0.4379494786262512, 0.8069830536842346, 0.34508004784584045, -0.34251052141189575, -2.0647802352905273, -0.1865817755460739, -1.1634469032287598, 0.38824373483657837, -0.4386087656021118, -1.0920497179031372, -1.6116926670074463, 0.6126755475997925, -0.11112818121910095, 0.3756486773490906, -2.5619189739227295, -1.2473363876342773, 1.5968401432037354, -1.2775225639343262, -0.03528537228703499, -1.3145123720169067, -0.29968494176864624, -0.08689973503351212, -0.45814085006713867, -0.4404234290122986, 0.26483267545700073, -1.194183349609375, -0.7066658735275269, 0.16334566473960876, 0.4257426857948303, 1.5231680870056152, -1.409101128578186, -0.322687029838562, -0.2699332535266876, -0.09930506348609924, 0.18209204077720642, 0.887733519077301, 1.0633282661437988, -0.7539107203483582, -1.0055272579193115, -0.4758121967315674, -1.3727192878723145, 0.5503652095794678, 0.9768749475479126, -0.8349646329879761, -0.4729483425617218, 2.612978219985962, 0.30680525302886963, -0.4971698820590973, -0.7752094268798828, 0.7029488682746887, 1.8085548877716064, 0.2338772863149643, 0.7034720182418823, 0.13381533324718475, -0.7494481801986694, -1.7108662128448486, 0.11301351338624954, -0.901695191860199, 0.3213406801223755, -0.6238200068473816, 0.5823501944541931, -0.5323470234870911, 0.3111174702644348, 0.8651257753372192, 0.9830660820007324, 0.492588073015213, -0.26943472027778625, 0.05797550827264786, 0.0683678388595581, 0.006284733768552542, -0.7973535060882568, 0.20253242552280426, -1.0061113834381104, -1.128639817237854, -0.3602786362171173, -1.5461117029190063, -0.8718245625495911, 2.7942423820495605, 0.5181755423545837, 0.4705814719200134, 0.46691203117370605, 0.7113851308822632, -1.229928731918335, -0.6437487006187439, 1.3173487186431885, -0.2616162896156311, 2.003291606903076, 0.39631137251853943, 0.9582318663597107, 0.4962838292121887, 1.201545000076294, -0.6473981142044067, -0.5774564743041992, -0.22537043690681458, 0.7956503629684448, -0.7322295308113098, -0.34919869899749756, -0.05217602849006653, 1.1545782089233398, -2.276101589202881, -0.13122332096099854, -0.0034940901678055525, 0.7198931574821472, 0.8583489656448364, 2.4641623497009277, -0.42919105291366577, -0.0699806734919548, -0.006874337326735258, 0.15676315128803253, -1.7546254396438599, -0.05051339045166969, -1.2858232259750366, -0.1788361370563507, -1.630370020866394, -1.9627692699432373, -1.2463942766189575, 1.1858021020889282, 0.6175103187561035], [0.5005456805229187, 0.2994251251220703, 1.084900975227356, 0.06655522435903549, -0.06277864426374435, -0.5227387547492981, 0.01537409145385027, 1.0663632154464722, 0.9641962647438049, 0.0674823522567749, -1.2971733808517456, -0.029529327526688576, 1.64556086063385, 0.1909942924976349, 1.5043591260910034, -1.3209595680236816, -0.14511819183826447, -0.26877132058143616, -0.09328631311655045, -1.8767369985580444, -0.20318885147571564, 1.485644817352295, 0.38528650999069214, -1.0872905254364014, -0.334878534078598, 0.795488715171814, 0.6302312612533569, 0.3748718500137329, 0.27638283371925354, -0.7050007581710815, -0.32471153140068054, 0.8859951496124268, 0.28182747960090637, -0.08361883461475372, -0.7908108234405518, 1.1521292924880981, 0.047841109335422516, 0.7916946411132812, -0.23419605195522308, -1.1433687210083008, 1.8739392757415771, 0.05233600363135338, -1.1404037475585938, -0.05481589213013649, 0.7533888816833496, -1.8324400186538696, -2.247317314147949, -0.5203990340232849, 0.6913854479789734, -0.3418886661529541, -0.15745027363300323, 0.07354051619768143, 0.08190706372261047, -1.1967904567718506, -0.34405550360679626, 0.173810675740242, 1.831252932548523, -1.3613636493682861, -1.2618783712387085, -0.037935443222522736, 1.799391269683838, 0.7293365001678467, 1.6452821493148804, 0.15475352108478546, -1.3032608032226562, -0.08969101309776306, 0.5927349328994751, 1.2029601335525513, 0.5270178914070129, 0.8727645874023438, 0.7394521236419678, -0.011697669513523579, 1.6218984127044678, 0.8901119232177734, 0.976426899433136, -1.0378737449645996, 0.9092933535575867, 0.09252388775348663, 0.9423288106918335, -0.3461318016052246, 0.7727918028831482, 0.24954582750797272, 0.47307538986206055, 0.6140487790107727, 2.490774393081665, 1.4239121675491333, -0.36205700039863586, 1.660085678100586, -2.053915500640869, 0.017761874943971634, 0.018726851791143417, 0.0436210110783577, -0.05708752200007439, -0.011002615094184875, 1.6897053718566895, 0.3647138476371765, 0.8657283186912537, 0.5496982932090759, 0.20613040030002594, -2.125006914138794, 0.5189468860626221, 2.2047035694122314, 0.05279180407524109, -0.5897909998893738, 0.7998209595680237, -0.418443888425827, -0.3024771213531494, -0.889897346496582, -0.1445009708404541, 1.0738507509231567, 0.25885245203971863, 0.8105308413505554, -0.790039598941803, -1.095589280128479, 1.8536731004714966, -0.2876759469509125, 0.5471369028091431, 1.2977559566497803, -0.10700860619544983, -0.9923677444458008, 0.15095680952072144, 0.8945554494857788, 1.0632531642913818, 0.18539844453334808, 0.5030174851417542, 1.9796831607818604, -1.3600313663482666, -1.5038871765136719]], [[-0.8420848846435547, -0.09758614748716354, 1.2607827186584473, -0.8873891830444336, 0.8984737992286682, 0.0084818871691823, -2.213167428970337, -0.16130894422531128, 2.546891689300537, -0.6623049974441528, 0.28400981426239014, -0.1367419958114624, 0.7697848081588745, -0.7753949165344238, -0.30699801445007324, -0.8405590057373047, -0.597633421421051, -0.70423424243927, 0.5244295597076416, 1.9064335823059082, 2.2764527797698975, -0.5465953946113586, -0.6608413457870483, 0.8844273686408997, 0.5874342918395996, 0.5502797365188599, 0.1427326500415802, -0.4891047179698944, 0.08562097698450089, -0.20481565594673157, 0.17535719275474548, 0.4210796058177948, 0.6031660437583923, -0.8092361092567444, 0.058742985129356384, -0.3230331540107727, -0.1289302110671997, 1.0963318347930908, 1.2947311401367188, -0.8426129221916199, -0.09068700671195984, -0.8092560172080994, -1.812424659729004, 1.1282132863998413, 1.0298528671264648, -1.9739185571670532, -0.2306968718767166, -0.43084442615509033, -1.2182576656341553, -1.5916160345077515, -0.04006842523813248, -0.33833929896354675, -0.44975998997688293, -0.8732576966285706, 1.6009571552276611, 1.049407958984375, -2.0680458545684814, 1.618605136871338, -0.47278061509132385, 0.3665520250797272, -0.30472373962402344, -1.25692880153656, -1.224169373512268, -1.0926587581634521, 0.8015680909156799, -0.7539668679237366, -1.7022271156311035, 1.0045278072357178, 0.4811982810497284, 1.7861480712890625, 1.0288970470428467, -0.26786378026008606, -0.44660937786102295, -0.0991138219833374, -0.3867066204547882, -1.4508912563323975, 0.992963969707489, -1.462127447128296, 0.9082863330841064, 1.1262218952178955, 1.265712857246399, -0.9873963594436646, 1.8553614616394043, 0.5987741351127625, 1.149707317352295, 1.1013543605804443, -1.2175564765930176, 0.5352296233177185, 1.5565245151519775, -1.8035862445831299, -0.2753455936908722, -0.8112193942070007, 0.475579172372818, 0.2638776898384094, 0.06660578399896622, -0.16554147005081177, -0.2599821984767914, -0.7564226388931274, -0.17710484564304352, -0.9905958771705627, -1.2642114162445068, -0.6800066232681274, -0.6585279107093811, -1.6619058847427368, 0.785428524017334, -2.084975242614746, 1.0211600065231323, -0.23686452209949493, 0.7529661655426025, 0.9153985977172852, 0.8031628727912903, -0.9532154202461243, -0.47939497232437134, -0.8804871439933777, 1.8683680295944214, -1.7674237489700317, 0.015239981934428215, 0.7173215746879578, 0.6598366498947144, -0.817791759967804, 0.7250207662582397, -1.210191249847412, -1.0414565801620483, 0.11044217646121979, -0.041022494435310364, 0.689744770526886, -0.05835268646478653, 0.06921112537384033], [-1.7698240280151367, -0.6002169847488403, -0.3099432587623596, -0.9189499616622925, -0.909144937992096, -0.8307490944862366, 1.2932591438293457, 1.6578941345214844, -1.7181050777435303, 1.2494608163833618, -0.32819774746894836, -0.07276920229196548, -0.863496720790863, 0.9310077428817749, 0.40589964389801025, 1.2422137260437012, -1.4870346784591675, -1.615709900856018, -0.7473857402801514, 0.544443666934967, 0.7801890969276428, -0.23611527681350708, -0.8058974146842957, -0.2778919041156769, -1.7369720935821533, -1.4776701927185059, 0.6786106824874878, -0.7073526978492737, 0.8514596223831177, -0.20987114310264587, 0.280300110578537, 1.5029774904251099, -0.024470586329698563, -1.8647098541259766, 0.23274531960487366, 0.0209425650537014, -1.1868940591812134, -0.21490001678466797, -0.5301893949508667, 1.5065141916275024, -0.5360124707221985, -0.6662110686302185, -0.9186500906944275, -1.121322751045227, -0.8182168006896973, 0.2826938033103943, -1.1239500045776367, -0.6977444291114807, 0.24906344711780548, 2.3032643795013428, 1.5798215866088867, 0.2063549906015396, -0.9783197641372681, -0.3038780093193054, 0.128486767411232, -0.7099384665489197, 0.7359831929206848, -0.07412280887365341, -1.155791997909546, -1.5274839401245117, 0.40253862738609314, -0.29749637842178345, 0.38843151926994324, -1.2594565153121948, -0.19984124600887299, 0.8908401131629944, -0.2211000919342041, -0.657515823841095, 1.5882489681243896, 0.4908711612224579, 0.6301643252372742, 2.285417318344116, -0.8438712358474731, -0.38906270265579224, -0.5925664305686951, -0.4609789252281189, 0.4464460611343384, 0.6482523083686829, 0.1265815645456314, 0.4634813964366913, -0.9072567820549011, 0.14294041693210602, -3.1491806507110596, 0.9859470725059509, -0.015206851996481419, 0.6098744869232178, -1.209574818611145, 0.4366934895515442, -0.8733066320419312, -0.30624374747276306, -0.8987082839012146, 0.33983755111694336, 1.1314610242843628, -0.8071130514144897, 1.2970143556594849, 1.3138363361358643, -0.8192815780639648, -0.05006493255496025, 0.8870099782943726, -1.0987590551376343, -0.840582549571991, 0.7097324132919312, -0.4517649710178375, 1.0342929363250732, 1.0197170972824097, -0.38414865732192993, 0.4348060190677643, -2.2234857082366943, 1.0176432132720947, -1.1873095035552979, -1.2917178869247437, -0.31421685218811035, 0.7534536123275757, 0.9492095112800598, -0.9043976664543152, -1.5086919069290161, 0.04243521764874458, -1.0805981159210205, -0.06463992595672607, 0.31552088260650635, -1.9013196229934692, 0.19416724145412445, -0.5414760112762451, -0.09997083991765976, 1.1921048164367676, -0.8131222724914551, 0.6327794194221497, 0.6764262318611145], [1.3880037069320679, -0.7286482453346252, 0.08478138595819473, -3.1630327701568604, 0.17347924411296844, 0.13074550032615662, -0.4813208281993866, -0.03680814057588577, -0.12719424068927765, -0.48812365531921387, 0.4592973589897156, -0.5910556316375732, -1.112735390663147, -0.3833763301372528, 0.060499630868434906, -0.2243874967098236, 0.9270383715629578, -0.31359443068504333, -0.6073119044303894, 0.736541748046875, 0.1992097645998001, -1.2217226028442383, 0.3368937373161316, -1.1065407991409302, -0.7077690362930298, 1.8728477954864502, 1.526082992553711, -1.538343071937561, 1.6703104972839355, -0.017450137063860893, -1.6155942678451538, 1.4665498733520508, -0.5038803219795227, 1.7004303932189941, 0.23409239947795868, 0.4871007204055786, -0.17907527089118958, -0.5249640941619873, 0.48502418398857117, -0.8218110203742981, 0.22552669048309326, -0.6355027556419373, 0.6643446087837219, 0.5136303901672363, -1.0184149742126465, -0.6242307424545288, 0.720434308052063, 0.5414584875106812, 0.890593409538269, -1.435341715812683, 1.230211853981018, 0.5342888832092285, -0.21113519370555878, 0.9461296796798706, -2.1832244396209717, 0.5974839925765991, 0.2468605935573578, -1.5421009063720703, 1.2194750308990479, -0.9994413256645203, 0.1776605248451233, 0.12846717238426208, 1.074323296546936, 1.4307702779769897, 0.7070430517196655, -0.287557989358902, -0.5944305062294006, 0.1338784545660019, -0.097408726811409, 0.4017373323440552, 1.429880976676941, -0.0918252170085907, 0.26557034254074097, 0.06256581097841263, 0.10472260415554047, 0.2774161100387573, -0.7124656438827515, 0.5021542906761169, 0.38215944170951843, -0.0980897918343544, -1.617550253868103, 0.006369838956743479, 0.515038251876831, -1.2044636011123657, -0.6191007494926453, -0.1402752697467804, 0.8944921493530273, -0.6775226593017578, -0.2857552766799927, 0.5730380415916443, 0.8984712958335876, 0.12162736058235168, 0.049248840659856796, 0.38914015889167786, 0.284706175327301, 0.07615168392658234, -1.771990180015564, 0.32260408997535706, -0.1685762256383896, -1.110885739326477, 1.4333223104476929, 0.3331683278083801, 2.0296225547790527, -0.2528766393661499, -1.1539843082427979, 0.4282091557979584, -0.4165807366371155, 1.89883291721344, 0.0503203310072422, 0.7634414434432983, -0.6250267624855042, 1.0820157527923584, 1.6562381982803345, -0.8226628303527832, -1.6098606586456299, 0.13288253545761108, 1.8232421875, 1.2247073650360107, -0.27963578701019287, 1.877388596534729, -0.5578170418739319, -0.27513259649276733, -0.19490176439285278, 1.191343069076538, -0.5205691456794739, 1.3777198791503906, 1.213720679283142, 0.41452065110206604], [0.4731965959072113, -1.0867650508880615, 0.36529213190078735, 1.2230587005615234, -0.8212100863456726, -0.9514446258544922, 1.7744332551956177, -0.3029860854148865, 0.9698521494865417, -0.1874585598707199, -0.3193581700325012, 0.5621806383132935, 0.5212012529373169, 0.5472068190574646, 0.6160128116607666, 0.23240084946155548, 0.5407387018203735, 1.7769430875778198, -1.3889245986938477, -1.2170113325119019, 0.17120087146759033, -0.7630832195281982, 1.7133821249008179, 1.4465715885162354, -0.6811662316322327, 0.001196400378830731, -1.0045517683029175, 0.5115078091621399, 0.7017806172370911, 0.35486242175102234, 0.2165067493915558, 0.8943942189216614, 0.007773916237056255, -0.30160972476005554, -0.9171001315116882, -0.3951599597930908, -0.8033998012542725, -1.2398895025253296, -0.850562572479248, 0.06511908024549484, 0.8644833564758301, -0.3006924092769623, -0.6139533519744873, -1.7486757040023804, 0.018716778606176376, 0.6583198308944702, 0.08877431601285934, -0.5339060425758362, 0.9999680519104004, 0.26368531584739685, 0.6157394051551819, -1.5826740264892578, -0.1294233202934265, 1.3273102045059204, 0.969474196434021, 0.6250964999198914, 1.6388890743255615, -0.8084178566932678, 1.9334913492202759, 0.8810744881629944, 0.06825607270002365, 1.3148423433303833, 0.018147069960832596, 0.12215610593557358, 0.005975441075861454, 0.527825653553009, 0.635041356086731, 0.06202269718050957, 0.24152915179729462, 0.5946447253227234, -0.16333536803722382, 0.24498476088047028, -1.7062103748321533, -1.406398892402649, 1.616599440574646, -0.19396834075450897, 0.4776898920536041, 0.533905565738678, -0.2615944743156433, 0.49535703659057617, 1.814194917678833, -0.5300115346908569, -1.4167850017547607, 1.3338030576705933, -0.34206998348236084, -0.7048787474632263, -0.7503555417060852, 0.26601144671440125, -1.2506474256515503, 1.266489863395691, 1.8448352813720703, -1.612960696220398, -0.49676525592803955, 0.5460910797119141, 2.6959290504455566, 1.6105682849884033, 0.9408723711967468, 0.04978606477379799, -2.2492592334747314, 2.3666481971740723, -2.195542097091675, -1.171596884727478, -2.755413055419922, 1.7388278245925903, -2.1433591842651367, -1.7899402379989624, -0.3255421817302704, 1.3062342405319214, 0.25757014751434326, 0.1874886304140091, 0.948116660118103, -0.191525399684906, 0.17745676636695862, -0.7424665689468384, 1.5246679782867432, -0.5643502473831177, -0.04720291495323181, 0.7721895575523376, -1.5148752927780151, -1.2196979522705078, 1.5450630187988281, -0.7197016477584839, 0.14107179641723633, -0.811038613319397, 1.4524004459381104, 0.11344946175813675, -0.38648471236228943, 0.15720854699611664], [0.15688583254814148, 0.8676983714103699, 0.7379401922225952, -0.6508821845054626, 1.499076247215271, -0.33715707063674927, 0.40157076716423035, -1.637588381767273, -0.4105222523212433, 0.7123782634735107, -1.811692476272583, 0.45347920060157776, -1.3672672510147095, 0.9011720418930054, 0.3513924479484558, -0.13238105177879333, 1.9090362787246704, 0.8582288026809692, 0.41413021087646484, -0.8323530554771423, 0.49028778076171875, -0.5662314295768738, 1.8326700925827026, 1.2693021297454834, -0.06841444224119186, -1.0353525876998901, -0.08622883260250092, 1.1646218299865723, 0.29491326212882996, 2.360954999923706, -0.5825190544128418, -0.3681677281856537, -0.59662264585495, 0.8431475162506104, 0.925986647605896, -0.7683084607124329, -0.06585367023944855, -1.0682814121246338, 0.4010612964630127, 1.1717936992645264, 0.9272884726524353, 0.11176304519176483, -0.6846615076065063, 0.259822279214859, 2.3714540004730225, 0.5392730832099915, -1.5813024044036865, -0.2465391457080841, -0.4041222631931305, -0.6293182373046875, -1.3121556043624878, 0.5258233547210693, -0.22791200876235962, -2.3816726207733154, 0.716383159160614, -2.0518136024475098, -0.2141348421573639, 0.8086969256401062, -0.04604475572705269, -0.8540214896202087, -1.2725025415420532, -0.7528305649757385, 0.5203066468238831, 0.06953626871109009, -0.3171895146369934, -1.9904670715332031, -0.5838937163352966, 0.5157138705253601, -0.4502476453781128, 0.5973765254020691, -1.5693634748458862, 1.4501713514328003, 1.894089698791504, -0.5419800877571106, -0.20154337584972382, -0.5338770151138306, 0.3383726179599762, 0.5663775205612183, 0.18268448114395142, 1.956101417541504, 0.32315897941589355, -0.30113527178764343, -0.8535177111625671, 1.0723251104354858, 0.5354126691818237, 0.07027027010917664, 0.09766378253698349, 0.9133367538452148, 1.8948934078216553, 1.495662808418274, -1.0006626844406128, -0.340192973613739, -0.008943283930420876, -0.5249000787734985, 1.388683557510376, 1.9008535146713257, -0.3174939453601837, 0.5137031078338623, 0.3942011594772339, -1.501886248588562, -0.32864347100257874, -0.6497868895530701, 0.08298242092132568, 0.6034612059593201, -0.1781555563211441, -1.0567907094955444, 0.5928325653076172, 1.2378697395324707, -1.602487325668335, -0.14381958544254303, -0.14128391444683075, 0.9893893003463745, 1.7476688623428345, 0.1572369784116745, 0.06242500990629196, 0.2605006694793701, -0.8901237845420837, 0.45395076274871826, 1.8904492855072021, 1.237243890762329, 0.4760359227657318, 0.7450711131095886, -0.3840675950050354, 1.255723476409912, 0.4980933964252472, 1.619434118270874, -1.3619709014892578, 0.7836014628410339], [-0.5929936170578003, -0.4086473882198334, -1.1973369121551514, 0.18272295594215393, 1.8450793027877808, -0.2116188257932663, 0.35257983207702637, -0.303671658039093, -1.1036367416381836, 0.27021268010139465, 2.0589354038238525, 2.7936272621154785, 1.2840933799743652, 0.09085545688867569, -0.9535512924194336, 0.22453010082244873, -0.26043790578842163, 1.1264207363128662, 0.8771411776542664, -1.7139091491699219, -0.7695232629776001, 0.9719972610473633, 0.5417147278785706, -0.6752691268920898, -1.157676100730896, 0.41181832551956177, -1.008621335029602, -0.35047852993011475, -1.0609527826309204, 0.022482438012957573, 0.8868164420127869, -0.47658947110176086, 2.286106824874878, -1.4638465642929077, -0.9280365109443665, 0.012581928633153439, 0.24325555562973022, -1.507981777191162, 0.10401810705661774, 0.929565966129303, -1.2707695960998535, -0.4098793864250183, -0.8057934641838074, 0.5486623644828796, -0.9728558659553528, 0.8088281154632568, 0.5139827728271484, 1.6533328294754028, -0.8114999532699585, -0.6563020348548889, -0.8743404150009155, 0.1640995740890503, -0.5815942883491516, -1.4269936084747314, -0.8927829265594482, 1.2544071674346924, 0.827164888381958, -1.2951761484146118, 0.8140358924865723, 1.8936457633972168, 0.021472038701176643, -1.0773868560791016, 1.5773649215698242, 1.636350393295288, -0.5358284711837769, -1.3603395223617554, -1.1595499515533447, 1.4484233856201172, 0.7984007000923157, 0.41014033555984497, -1.299822449684143, 1.1489028930664062, -1.5943254232406616, 0.10039142519235611, -1.1696622371673584, -0.37257200479507446, 0.7116169333457947, -1.0479869842529297, 1.8610011339187622, 0.38091278076171875, -0.40573441982269287, 0.06871002167463303, 0.5284467339515686, 1.2028839588165283, -0.5604866743087769, 1.1695681810379028, -1.4045593738555908, 1.2842061519622803, -0.1360769271850586, 0.2921653091907501, 0.2510376274585724, -0.16721892356872559, 1.265357255935669, -1.9981331825256348, -1.4138858318328857, -0.11185789108276367, 0.05700811743736267, -0.802988588809967, 1.3478705883026123, -0.9874293208122253, -0.6767877340316772, -0.41551434993743896, -0.47746872901916504, -0.9758047461509705, -0.47397372126579285, -0.21056273579597473, 0.4355272650718689, -0.8015840649604797, -0.5527912974357605, 0.15298990905284882, 0.025336917489767075, -1.237730860710144, 0.06307942420244217, -0.8325855731964111, -1.7948330640792847, -1.9253968000411987, 1.4686378240585327, 0.25529035925865173, -0.8753595352172852, -1.488128900527954, -1.7767430543899536, -0.5083788633346558, 0.8854390978813171, -0.872118353843689, -1.8014417886734009, 1.3408340215682983, 0.814347505569458, 0.9648537039756775], [-0.11052977293729782, 0.7790848612785339, -1.1878917217254639, -1.4519544839859009, -0.7725050449371338, -0.33597904443740845, -0.0420980341732502, -0.3004678189754486, -0.998908519744873, -1.5338971614837646, -0.3258686661720276, -2.1590824127197266, 0.15752550959587097, -0.3792541027069092, -2.4851231575012207, 0.7415242791175842, -0.16968762874603271, -0.4673304557800293, 0.0779290422797203, 0.7149803042411804, 0.5112883448600769, 0.6386305093765259, -0.0926642194390297, 0.835056483745575, -0.8058256506919861, -0.3747568130493164, -0.11548162251710892, -0.7932275533676147, 0.5059810280799866, 1.3944520950317383, -1.535732388496399, -0.20466117560863495, 0.813981831073761, 0.9837263226509094, 1.42547607421875, -0.7665818333625793, 0.30361396074295044, 0.3773733973503113, -1.3642165660858154, -2.038102626800537, 0.37132135033607483, -1.2922488451004028, 0.09501030296087265, -1.781113624572754, -0.3654947876930237, 1.1893292665481567, -0.7194201946258545, -0.3439497947692871, 0.02385762706398964, -1.5165445804595947, 1.5440959930419922, 1.6431249380111694, 0.6312540173530579, 0.07950767874717712, 0.24947631359100342, 0.943640947341919, -0.5331088900566101, 1.2523853778839111, 1.3411610126495361, 1.1928372383117676, 0.06231299042701721, -1.7093337774276733, -0.6874746084213257, 0.029658610001206398, 0.37355276942253113, -0.17765358090400696, -1.1043360233306885, -0.8822921514511108, 0.015971390530467033, -1.7288854122161865, 0.689810037612915, -1.0021957159042358, 0.2536175847053528, 0.2344314008951187, 0.031845804303884506, 1.3530622720718384, 0.6716488599777222, -1.4826812744140625, 1.7379416227340698, -1.4199084043502808, -0.1518356055021286, 1.0281929969787598, 2.0808119773864746, 0.6220405697822571, -0.7145789265632629, 0.2686893045902252, 0.36865201592445374, -0.6488391757011414, 1.6343698501586914, -0.3956291675567627, -0.9513610601425171, 0.42468008399009705, 1.9862140417099, -0.7468408942222595, 0.31751543283462524, -0.4721819758415222, -0.20152971148490906, -1.7391897439956665, 0.0765770748257637, -1.6001160144805908, -1.046441912651062, 0.5922218561172485, 1.0386450290679932, 0.35836783051490784, 0.14602892100811005, -0.6350112557411194, 0.36258938908576965, 0.11749482899904251, 0.07251396775245667, 0.10954247415065765, 0.025561079382896423, 0.09631967544555664, -0.15594036877155304, -0.5407061576843262, 0.8875080943107605, 0.8932006359100342, -0.2725006937980652, -1.032094120979309, 2.199944257736206, 1.073548674583435, 1.335497498512268, -0.3522357642650604, -1.391905426979065, 1.4018901586532593, -0.3840349316596985, 1.8272048234939575, 0.7310769557952881, 1.3132951259613037], [-0.3534640967845917, 0.5506928563117981, 1.1235454082489014, 0.7723747491836548, 1.49750816822052, 1.1193054914474487, 0.1317436546087265, -0.6732020378112793, -1.6070880889892578, 0.7312780618667603, -1.4233015775680542, -0.7225989103317261, 0.5552958250045776, 0.5117445588111877, -0.3178001046180725, 1.0921601057052612, 3.1501381397247314, -0.8879659175872803, 0.8458857536315918, 1.0506194829940796, 0.04099831357598305, -1.06455659866333, 1.6307098865509033, -1.6741479635238647, -0.8539771437644958, -1.2646079063415527, 0.4172605872154236, 0.0684901773929596, -0.03000972792506218, -1.172243356704712, 0.9902759194374084, -1.5888147354125977, -1.275473952293396, -1.4930980205535889, -0.6444542407989502, 0.9858022928237915, -1.4077881574630737, -0.8724355697631836, -0.23416642844676971, 0.6185197830200195, 0.7345200777053833, -0.7486121654510498, 0.9962650537490845, 1.015366792678833, 0.4089900255203247, 0.1617588996887207, 1.2650278806686401, 0.11504939198493958, 1.7174044847488403, -0.38754037022590637, -0.9067336320877075, 1.1798747777938843, -1.3737565279006958, 1.5725421905517578, 0.5992121696472168, -1.090794324874878, -0.7959438562393188, -0.5422112345695496, -2.218442678451538, 0.21256008744239807, -2.1320667266845703, 0.8741200566291809, -1.2423442602157593, 3.269934892654419, 0.7184888124465942, -0.6689645051956177, -0.7874033451080322, 0.30443161725997925, -1.8188538551330566, -1.5010788440704346, 0.41210508346557617, -0.2294624149799347, 1.917935848236084, 0.08057639747858047, 1.3163319826126099, -0.4080986976623535, -0.3874613046646118, 1.0392200946807861, -0.30932286381721497, -2.3180744647979736, 0.828747570514679, -0.3657194972038269, -0.08617417514324188, 0.585940957069397, -0.7243250012397766, -0.7303438782691956, 1.989211916923523, 0.1638580858707428, 1.018198013305664, 0.14816179871559143, -0.5769339799880981, -1.3977471590042114, 0.4421417713165283, -0.3597002625465393, -1.4861196279525757, -0.8630276322364807, 1.189535140991211, -0.038762036710977554, -1.054280400276184, -0.04032985493540764, 0.4856022894382477, -0.4196805953979492, 0.9819391965866089, 2.0133204460144043, -0.667819082736969, -0.41872715950012207, -0.09216660261154175, -0.8794824481010437, -0.5161119103431702, 0.06579501926898956, -0.3247211277484894, -0.3245902359485626, 0.9007464647293091, -0.42293524742126465, -0.38609597086906433, -0.8948697447776794, -0.20168964564800262, 0.2765231430530548, -0.3241818845272064, -0.875701904296875, -0.7813073992729187, 0.8005658984184265, -0.37311041355133057, 0.2561577558517456, -1.2017031908035278, -0.971924364566803, 1.2558097839355469, -1.0818555355072021]], [[0.7357699275016785, 1.0904830694198608, -0.03110329806804657, -0.2440067082643509, -0.7500715255737305, 0.8548301458358765, -0.47307097911834717, 0.37417900562286377, -0.7408764958381653, -0.05456779524683952, -0.9821315407752991, 0.2503346800804138, 0.2031196504831314, -0.9373407363891602, -0.4459565579891205, -1.1164416074752808, -0.6686015725135803, 0.23713237047195435, -0.16707095503807068, -0.5501143336296082, -0.36715927720069885, 0.13603152334690094, -0.019841961562633514, 0.7425075173377991, 0.551895260810852, -0.1310945749282837, -0.5705747604370117, -1.0355963706970215, -0.7079426646232605, -1.0734399557113647, -2.471585750579834, 1.2504143714904785, -0.49551549553871155, -2.215627908706665, -1.007020115852356, -1.8747258186340332, 1.8569567203521729, -0.7710604071617126, -1.910239815711975, 1.005190134048462, -1.1435788869857788, -1.6729154586791992, 0.31840768456459045, -1.3239407539367676, 0.678330659866333, -0.43894174695014954, -0.18502050638198853, -0.18550212681293488, -0.22166745364665985, 0.08529116958379745, 1.342283010482788, 1.035774827003479, 1.0185351371765137, -0.2874176502227783, -1.6801369190216064, -0.6837306618690491, 0.5319011211395264, 1.533512830734253, 0.7764478325843811, 0.7325175404548645, 2.7793233394622803, 1.3313663005828857, 1.2972655296325684, -0.11173848807811737, 0.2830376923084259, 1.7536616325378418, 0.7329756021499634, -0.8542625308036804, -0.07838529348373413, 0.32823729515075684, -1.1645383834838867, -0.022218389436602592, -0.18207111954689026, 0.9338870048522949, -1.983712911605835, -0.5123054385185242, -0.7872235178947449, -0.979366660118103, 1.172134518623352, 0.7644405364990234, -0.21636618673801422, 0.14567047357559204, -0.08128238469362259, -0.4498523771762848, -1.047218680381775, -0.33428263664245605, -0.20402753353118896, 1.0628297328948975, -0.5033755898475647, -1.674970269203186, 0.6813990473747253, 0.6394944190979004, -0.6449286937713623, -0.692521333694458, 1.1768535375595093, 0.6321377158164978, -1.3297629356384277, 0.7157272696495056, 0.2667594254016876, 0.45217734575271606, 0.27487578988075256, -0.6339762806892395, -0.632569432258606, 0.3710382580757141, -1.0962226390838623, -0.575534999370575, -0.9321308135986328, -1.156416654586792, 0.17084327340126038, 0.20721617341041565, -0.09369999170303345, -1.6332635879516602, -0.8329198360443115, -0.4293510317802429, -0.1885569542646408, -0.5987759828567505, 2.2198033332824707, 0.9048826098442078, 2.012489080429077, -0.5688177347183228, -0.213300421833992, 0.7364381551742554, -0.8103317022323608, -0.5942371487617493, 0.7053214311599731, -1.316866159439087, 2.2854764461517334, 0.7276705503463745], [-0.21667197346687317, -1.4982364177703857, 1.1573065519332886, -1.8326376676559448, 0.9344457387924194, -0.052143391221761703, 1.5652904510498047, 0.078682541847229, 1.5514706373214722, -0.5745064616203308, 0.35670074820518494, 0.23292046785354614, 1.269511342048645, 0.47886931896209717, -1.4324471950531006, -0.09068839251995087, -0.35728082060813904, 0.5355604887008667, -1.708418846130371, 0.5508553385734558, 0.6624270677566528, -0.08977367728948593, -0.16126205027103424, -0.5027607679367065, -0.2522030174732208, 0.9362190365791321, 0.8580822348594666, 0.17861954867839813, 0.30062368512153625, 0.7095088362693787, 0.19295941293239594, -0.5358613729476929, 0.5027425289154053, -0.6315310597419739, 0.5483578443527222, -0.021777868270874023, -0.3680333197116852, 0.8856121897697449, -1.3765469789505005, -0.3804906904697418, -0.4378712475299835, 0.47905129194259644, 1.6110347509384155, -0.014578523114323616, -0.6309909224510193, -1.3753947019577026, -1.8654346466064453, 0.929226279258728, 1.1642659902572632, 0.26404672861099243, -0.15861552953720093, -0.08115965127944946, 0.3641672432422638, -0.05898623540997505, -2.14520001411438, 1.8476083278656006, -0.3528803586959839, 0.8571426868438721, 0.602358877658844, 1.0729737281799316, -0.9407114386558533, -0.9443584680557251, 2.2330687046051025, 0.13846665620803833, 1.0306819677352905, -0.8109065890312195, -1.3994845151901245, 0.47944751381874084, -1.069778323173523, 0.6419216990470886, 0.19814151525497437, 0.8195075988769531, -0.47260528802871704, 0.5989230871200562, -0.24301022291183472, -0.4018992483615875, 0.11804737895727158, -0.14852669835090637, -0.07741352170705795, 1.6016947031021118, 0.21780110895633698, 1.2924349308013916, -0.7382308840751648, -2.44918155670166, -0.7485328316688538, 0.519987165927887, 0.24944961071014404, 0.5109305381774902, 1.28333580493927, -0.9269894361495972, 0.621631383895874, 0.502648115158081, -0.0856735110282898, 1.7677583694458008, -0.6351884007453918, 0.5821057558059692, -1.4520039558410645, 1.1866681575775146, -1.5464197397232056, 0.7160801291465759, 1.1887038946151733, -0.8117077350616455, -1.5910046100616455, -1.5032232999801636, -0.6420842409133911, 0.022651419043540955, 0.41801121830940247, -0.16367091238498688, -0.05706227943301201, -0.17155082523822784, 0.47855380177497864, 0.4388306736946106, -0.4133893847465515, 0.31325656175613403, 0.7962421178817749, 0.48582661151885986, -0.5133364200592041, 0.2313324362039566, -0.7593621611595154, -0.5981859564781189, -1.1800440549850464, 0.24464499950408936, 0.3801167905330658, -1.6162679195404053, 0.3714723587036133, 0.01535211130976677, -1.5316702127456665, -1.5304712057113647], [-0.21499153971672058, -0.37589868903160095, 0.36480262875556946, -0.6623382568359375, -0.8572933673858643, -0.4748615026473999, -0.11328766494989395, 1.2090821266174316, -0.2136089950799942, 0.05110989138484001, -0.8486298322677612, 0.19468161463737488, -0.899580180644989, 1.0786136388778687, 0.6073864698410034, 1.6290339231491089, -1.0690820217132568, 1.1931064128875732, -0.5624408721923828, -0.007851239293813705, -1.2877743244171143, 0.8242811560630798, -1.0135389566421509, 0.13527071475982666, -1.6977177858352661, -1.4478658437728882, 0.1378214806318283, -0.14745110273361206, -0.07991506904363632, 2.4036192893981934, -0.47199270129203796, -0.9301869869232178, 1.8252202272415161, -0.010938052088022232, -1.2856471538543701, -0.6576066017150879, -0.18953044712543488, -1.054380178451538, -1.2470453977584839, -0.5180413126945496, 1.5269098281860352, 1.6813722848892212, -1.5089619159698486, 0.1017148494720459, 1.3497709035873413, 1.3858585357666016, -1.6747902631759644, 0.29625314474105835, 0.30873599648475647, -0.10592266917228699, -0.40801337361335754, -0.03347482532262802, 0.011064905673265457, -1.537006139755249, 1.6347661018371582, -0.37073060870170593, -0.38085880875587463, 1.4233160018920898, 0.43474188446998596, 1.749423623085022, 2.819269895553589, 0.11679580062627792, 0.6262811422348022, -0.6424574255943298, -0.1722387671470642, -0.7534509301185608, 0.8172225952148438, 0.7101467847824097, 1.759626865386963, 0.0838392898440361, 1.037529468536377, 0.9862056970596313, 0.2580699622631073, -1.1363022327423096, 1.0030651092529297, -0.47256895899772644, 0.17929242551326752, 0.7296721935272217, -0.5070073008537292, 0.5911109447479248, -0.6477540731430054, -0.538878858089447, -0.7851719260215759, 0.1779886931180954, -0.05939965695142746, 1.2115613222122192, 0.2848931550979614, 1.145115613937378, 0.366521954536438, -1.0336570739746094, 0.03785174340009689, 1.473325490951538, 0.8882237076759338, -1.0848227739334106, -1.0056591033935547, -0.3713291585445404, 1.1548011302947998, 1.5227831602096558, -0.7955714464187622, -0.504295289516449, 0.2675340175628662, 1.3831384181976318, -1.5805734395980835, -0.9412325024604797, -0.1822347193956375, -0.04175037145614624, -1.1280981302261353, -0.2636929452419281, 0.6977700591087341, 0.10289429873228073, -1.298933744430542, 0.7595588564872742, -0.7999964952468872, -0.3444979190826416, -2.2879862785339355, 0.7340718507766724, 0.7018710970878601, 0.8758835196495056, 0.32883134484291077, -0.2066284716129303, 0.42671093344688416, -0.6298395991325378, 0.4849950969219208, 0.0755828395485878, 0.1782972812652588, -0.22473646700382233, -0.05020894482731819, -0.6232871413230896], [-0.7672621607780457, 0.7920758128166199, -0.3343486487865448, -1.4473676681518555, -1.1776413917541504, -0.9435350298881531, -0.3087024390697479, -1.3240858316421509, -0.47385066747665405, 0.06706048548221588, 1.4533469676971436, 1.0240850448608398, 0.9155077338218689, 0.5488210320472717, 0.35411569476127625, 1.6948485374450684, 2.3290913105010986, -0.9357835650444031, 0.11074844747781754, -0.5981621146202087, -0.08775532990694046, -1.0862208604812622, -1.7074073553085327, -1.6378653049468994, 0.037707533687353134, -0.9740980267524719, -0.10460428148508072, 0.30097082257270813, -1.0380945205688477, 0.8027545213699341, 0.6529077887535095, -0.11631356924772263, -1.2523775100708008, -1.5872551202774048, -1.3880412578582764, 0.605209469795227, 1.240347146987915, -0.6626086235046387, -1.8325773477554321, 0.4247191846370697, -0.21249915659427643, 0.7631049752235413, 0.22593623399734497, -0.31253471970558167, 0.44965407252311707, 0.3906552195549011, -0.48306557536125183, -1.1633093357086182, 2.155402421951294, 1.4378879070281982, 0.0055229938589036465, 0.6084895730018616, 0.7691371440887451, -2.8471732139587402, -0.17688144743442535, 0.14473192393779755, -1.8153961896896362, 0.8294507265090942, -1.7420017719268799, -1.2664241790771484, 0.2788275182247162, 0.6019247174263, 1.1085602045059204, -0.5842787623405457, -0.9650906920433044, 0.5532054901123047, -0.8737550973892212, -0.3250502943992615, 0.2581879794597626, -1.2963391542434692, -0.5484864711761475, 1.366034984588623, 0.4338098168373108, 0.21904847025871277, 0.06427119672298431, 0.4049180746078491, -0.09838835150003433, -0.5108232498168945, -0.26274511218070984, 0.3796849846839905, -1.3151756525039673, 1.643480658531189, -0.13728965818881989, -0.8201581835746765, -0.07565434277057648, 2.046268939971924, -0.6110166311264038, -0.5749373435974121, 0.30199581384658813, -0.5562816262245178, 1.515735149383545, -0.9307144284248352, -0.047420043498277664, 0.9653275609016418, 0.3386927545070648, 1.4813176393508911, -0.949451744556427, -1.9666593074798584, 0.0339336022734642, 0.03436415269970894, 1.8954719305038452, 0.3960736393928528, 1.5373454093933105, 0.1572318971157074, 0.6078492403030396, -0.7976164817810059, 0.04721124470233917, -0.21240366995334625, 0.4843396842479706, -0.8810673356056213, -0.9154780507087708, -1.1459007263183594, -0.310456782579422, -0.9512798190116882, 0.7699844837188721, 0.3335041403770447, -1.1256788969039917, 0.17045477032661438, 0.6431204676628113, -2.426497220993042, 0.004571934230625629, 0.2761671841144562, -0.08466416597366333, -0.7016350626945496, -1.1748994588851929, 1.0169544219970703, 0.054380737245082855, 0.9391920566558838], [1.4712917804718018, 1.228007197380066, -1.9264110326766968, 1.561949372291565, 0.3952578902244568, 0.8852284550666809, -1.4104437828063965, -1.3875328302383423, 0.56144779920578, -0.16890272498130798, 0.4632323086261749, 0.4849758744239807, 0.6754207611083984, 1.1928753852844238, 0.4301126003265381, 0.6258671879768372, 1.6106237173080444, 1.1491116285324097, -1.638720154762268, -0.5988110899925232, 0.5633618235588074, 1.984600305557251, 1.6862914562225342, -0.7970185279846191, -0.017086608335375786, -0.8299508094787598, 1.5393303632736206, 2.2166032791137695, -1.1239746809005737, 0.8004220724105835, -1.4060959815979004, -0.17536351084709167, 2.247100591659546, -1.0090426206588745, 0.5583533644676208, -0.3171921372413635, 2.181692361831665, 1.3495944738388062, 0.09815158694982529, 0.8954665064811707, -0.5946269631385803, 0.4571567475795746, 0.23074384033679962, -0.343680202960968, -1.2571439743041992, 1.193963646888733, -0.715183675289154, 0.9801735281944275, -0.21128053963184357, -0.8346067667007446, -0.251603901386261, -0.9535520076751709, -0.3732675313949585, -0.07879116386175156, 0.5920276045799255, 0.33048972487449646, 0.2728235423564911, -1.485854148864746, 0.21572551131248474, 0.13134917616844177, -0.041598841547966, 1.242068886756897, -1.7405650615692139, 0.4512943923473358, 0.8038247227668762, -0.46500757336616516, 1.2758302688598633, 2.2694509029388428, 0.45966124534606934, 0.7199098467826843, 0.38986265659332275, 1.1205447912216187, -0.01610645279288292, -0.1358117312192917, 1.0392358303070068, -0.9353829622268677, 0.28584516048431396, 0.5143327713012695, -2.1869254112243652, -0.655604362487793, 0.3876691460609436, -1.5044350624084473, -1.2979472875595093, -1.4016774892807007, -1.6436047554016113, 0.6207820177078247, 0.09855901449918747, 0.9596302509307861, -0.760512113571167, -0.9733083844184875, -1.7276970148086548, -0.4410133957862854, -0.4677557647228241, -0.2577035129070282, -1.7914152145385742, 0.9823541045188904, -0.3743736147880554, -0.0033297017216682434, -0.01068541407585144, -0.7927822470664978, 0.8714592456817627, 0.4409175217151642, 0.05337359011173248, 1.4168879985809326, -0.21388299763202667, 0.6780398488044739, -0.3803437352180481, -0.1295144259929657, -1.430099606513977, 0.4165589511394501, 0.7193570137023926, -0.742953896522522, -0.19712843000888824, 0.2780148684978485, 0.24069954454898834, -0.16252878308296204, -0.7730544209480286, -0.06218560039997101, 0.3202528655529022, 0.8072916865348816, -0.6024262309074402, -1.5513403415679932, 0.30939772725105286, -0.11902624368667603, -0.6712594628334045, 0.8566460609436035, -2.270343065261841, 1.0179194211959839], [0.8678141832351685, 0.6102228760719299, 0.47815442085266113, 0.3255605697631836, -0.3262670040130615, 1.0662140846252441, -0.3199586272239685, -0.7421023845672607, -0.8938431143760681, -0.3790692687034607, -0.5054024457931519, -0.9819528460502625, -0.8088545203208923, -0.49236202239990234, 0.4969423711299896, 0.5151785612106323, 0.045899756252765656, -1.066981554031372, -2.4069371223449707, -0.9572281241416931, -0.48329460620880127, 1.9419759511947632, -0.27432024478912354, 0.06294135749340057, -0.9445936679840088, -0.43276864290237427, -1.1611404418945312, -0.36485201120376587, -1.0251407623291016, 0.07132913172245026, -0.7808430790901184, 0.601526141166687, 1.1952213048934937, 0.3559022545814514, -1.16942298412323, -0.5184974074363708, -0.12827496230602264, -0.886860728263855, -0.44839033484458923, 0.4090118110179901, -0.846302330493927, 1.9405725002288818, -0.7044844031333923, -0.1905783861875534, 0.2469828575849533, 0.07697148621082306, 1.9369823932647705, 1.0748226642608643, 0.8264045715332031, 0.735159158706665, 0.2802676260471344, -0.7955257892608643, 0.5056875348091125, 0.10143061727285385, -1.0600433349609375, 0.1622985303401947, 1.045936107635498, -0.05277867242693901, -1.933205246925354, 1.7691843509674072, 0.08284562826156616, 1.260591745376587, 0.35258403420448303, 1.6134546995162964, 1.7467268705368042, 0.5665335059165955, -0.09853067249059677, -0.6761158108711243, -1.0027155876159668, -0.11298616975545883, 0.39314568042755127, 0.78481525182724, -0.4240347445011139, 0.10466225445270538, 0.06568888574838638, 0.03807449713349342, -1.8925604820251465, 0.735242486000061, 0.7310463786125183, -0.0999516174197197, 0.5930948257446289, -0.597805380821228, 1.1185749769210815, 0.707107663154602, -0.18151548504829407, 1.2175755500793457, -0.061082638800144196, 0.6380906105041504, 0.502998948097229, -0.1316649615764618, 0.07710005342960358, -0.9768017530441284, 0.9114539623260498, -0.5114547610282898, 0.6991387009620667, -1.2890833616256714, 0.2725520730018616, 0.515815794467926, 0.9946267008781433, 0.07237087190151215, -0.5707104206085205, -0.5228750705718994, -0.0713188573718071, 1.0265482664108276, -2.871929407119751, 1.8135203123092651, 0.7152958512306213, -0.06181924417614937, 2.3221259117126465, 1.6356406211853027, -0.060637056827545166, 0.9716554284095764, 0.20679859817028046, 0.19759055972099304, 0.7280850410461426, -0.13144010305404663, 0.8434886336326599, 2.340050220489502, -1.2841359376907349, -1.2571865320205688, 0.33241167664527893, 1.302995204925537, -1.7938807010650635, 0.22400856018066406, 0.5659070611000061, 0.5996412634849548, -0.33220481872558594, 2.0059309005737305], [1.160386085510254, 0.551159679889679, 1.5331684350967407, 1.2904510498046875, -0.04728453606367111, 0.11420492827892303, -0.939450740814209, -0.24055010080337524, -1.469459056854248, -0.3896346390247345, 0.10121804475784302, 0.15678709745407104, 0.7316163778305054, -1.2990480661392212, -0.3389241099357605, -0.19648601114749908, -0.7857059836387634, -2.474781036376953, -1.6429474353790283, -2.5671093463897705, 1.880438208580017, -0.6290431618690491, 0.43209755420684814, -0.48773542046546936, -0.7402839064598083, -1.2896555662155151, 0.8827189803123474, 1.1141573190689087, 1.1548919677734375, 1.1097369194030762, -1.4954475164413452, 0.7254267930984497, -0.5690135955810547, 1.093239188194275, -0.1971811056137085, -0.21781067550182343, 0.5511006712913513, -0.5639526844024658, -0.5136721134185791, 0.7284673452377319, -0.030816202983260155, -1.2221299409866333, 0.04260030761361122, -1.0098479986190796, -1.1784170866012573, -1.0442594289779663, 1.0678956508636475, -0.2133679836988449, -0.7770057320594788, 0.28731369972229004, 1.4648770093917847, -0.6291236281394958, -0.7336247563362122, 0.33040863275527954, -0.3604060113430023, 1.5030523538589478, -0.19873173534870148, 0.19167816638946533, -1.754050374031067, 0.12267675995826721, -0.07159776240587234, -3.167224645614624, -1.989223837852478, 0.5725073218345642, -0.9968472719192505, -0.34559163451194763, -0.686371386051178, -0.7060654759407043, -0.16181381046772003, 0.48236650228500366, -0.11878208816051483, 0.3849036395549774, 0.25237053632736206, 0.7423004508018494, -0.4828297197818756, 1.8502284288406372, 0.005744041409343481, 2.7833073139190674, 1.1944462060928345, -0.04055139422416687, -0.3090095818042755, 0.18097788095474243, -1.5468316078186035, 1.0755881071090698, 0.1082935631275177, 1.1868572235107422, -0.858586311340332, 1.1780847311019897, 0.9032089114189148, 0.029540153220295906, 1.3594729900360107, -0.02711435593664646, -0.3646317422389984, -0.17767392098903656, 0.7898431420326233, -1.3844780921936035, -0.09276995062828064, 2.0071611404418945, -0.5258895754814148, 1.0857573747634888, -0.8592900633811951, -0.7325100302696228, 0.21762292087078094, 0.7653759121894836, 0.4170992970466614, 0.6765270829200745, 1.0437321662902832, 0.18435890972614288, -0.5692761540412903, -0.8069449067115784, 0.13916276395320892, -1.1479283571243286, 0.6639373898506165, -0.7481284141540527, 0.31810539960861206, -0.5325844883918762, 0.04735066369175911, -1.0948926210403442, 1.075990080833435, -0.380719929933548, -0.6027491092681885, -1.7757518291473389, -1.7533271312713623, -0.6594346761703491, -1.7261877059936523, 0.5887208580970764, -1.0070244073867798, -0.4220935106277466], [-0.15230949223041534, -0.12397617846727371, 1.2152835130691528, 0.291258305311203, -0.16516606509685516, 0.6762241721153259, 0.6411278247833252, -1.0697789192199707, -0.46540573239326477, -0.5483875274658203, -0.3981491029262543, -0.1984715759754181, 1.0354353189468384, -0.6478352546691895, -0.3156134784221649, -0.19188764691352844, 1.4522167444229126, -0.7852703332901001, 0.22869873046875, -0.15586410462856293, -0.5692543983459473, -0.2381134033203125, 0.14703300595283508, 2.1103248596191406, 0.12343473732471466, 0.27213752269744873, 0.006160991732031107, 0.3662371337413788, 2.081058979034424, 0.3835207521915436, -1.2131109237670898, -0.7866694927215576, -0.5515573024749756, 1.2314462661743164, 0.1904875487089157, -1.7243683338165283, 0.20536643266677856, -2.176129102706909, -0.02928897924721241, -0.2377062439918518, 0.683018147945404, -0.8551360368728638, 1.1234972476959229, -0.23143160343170166, -0.6040008664131165, -0.5306875705718994, 0.26086875796318054, 0.2838079631328583, -0.9387985467910767, 0.4789108335971832, 0.9092509746551514, -1.007232666015625, -0.16806429624557495, 0.27381452918052673, 0.5230271816253662, -0.23830769956111908, 1.6897552013397217, 1.2708626985549927, 0.21648819744586945, -0.7013816237449646, 0.9255144596099854, -0.3522382080554962, 0.4737650752067566, -1.261213779449463, 0.500798761844635, -1.6845650672912598, 0.08282078802585602, -0.2727496325969696, 0.7461093664169312, 0.27439868450164795, 0.08745261281728745, 1.6175642013549805, 0.13389013707637787, -0.8945814967155457, 0.49994587898254395, -0.23084606230258942, 1.3386656045913696, 0.586076021194458, -2.3904542922973633, -0.7631139159202576, -1.482740044593811, 0.3121435046195984, 1.8246476650238037, 1.1013784408569336, -0.4146876931190491, 0.2336399108171463, 0.48389747738838196, -0.7627175450325012, -1.1198694705963135, 0.1397174447774887, 0.9622180461883545, 0.23293063044548035, 0.44286492466926575, -1.9230648279190063, 0.37234795093536377, 0.5030712485313416, -1.0280451774597168, 0.2586556673049927, 0.317840576171875, -1.2473652362823486, -0.37589263916015625, -0.12960900366306305, 0.2857513129711151, 1.1280338764190674, -1.0242422819137573, 0.44233959913253784, 1.1748266220092773, 1.1433645486831665, 0.5612683296203613, 0.3821650743484497, 0.013905242085456848, -0.8943567276000977, 1.3101128339767456, 0.25085675716400146, -0.1567549854516983, -0.5176668167114258, -1.49082350730896, -0.3668157458305359, -0.9130109548568726, 0.7125357389450073, -1.4013152122497559, 0.22679460048675537, 1.3625003099441528, 0.3383302688598633, -0.2687436044216156, 2.33735728263855, 0.00721735879778862, -0.23625126481056213]]], [[[-0.046442121267318726, 0.49960771203041077, -1.0164427757263184, 0.26023560762405396, 0.14204570651054382, 0.5747970342636108, 0.2262447327375412, 0.5444238185882568, 0.03892834857106209, -1.4715548753738403, 0.8655656576156616, -1.2579511404037476, 0.4462779760360718, 0.3000680208206177, -1.3628506660461426, -0.03895392268896103, 1.1374765634536743, -0.7001190185546875, 0.531381368637085, 0.2340056449174881, 0.39737576246261597, -1.9269120693206787, 0.19486363232135773, 0.3154875338077545, -0.6498740315437317, 0.18261396884918213, -0.3737736642360687, 1.5678516626358032, 0.49455025792121887, 2.301884889602661, 0.35638228058815, -1.445081114768982, 0.7589585781097412, 1.9480113983154297, 0.44025713205337524, 2.106112480163574, 2.7986700534820557, -0.3111236095428467, -0.6519071459770203, -0.8161066770553589, -0.6410802602767944, 0.8021090626716614, -0.3179396688938141, 1.1849814653396606, -2.0701167583465576, 0.5070103406906128, -0.3332926332950592, -0.13856585323810577, -0.146039679646492, 1.7852599620819092, -0.92917400598526, -0.2935740649700165, 1.5946910381317139, -1.0352783203125, 0.8971198201179504, -1.7833060026168823, 0.7313882112503052, 1.0177714824676514, 0.4058762192726135, 0.9472557902336121, -1.086724042892456, -0.372516930103302, 1.0619550943374634, -1.3864160776138306, -1.486702799797058, -0.18848064541816711, 0.9598629474639893, -1.1202818155288696, 0.8254013061523438, 1.124529480934143, -1.4380204677581787, 0.3071911334991455, 1.6663583517074585, 0.8976930379867554, 1.0481849908828735, -0.5111175179481506, -2.7666285037994385, -0.5788179039955139, -0.1838410347700119, -1.801068902015686, 0.8077029585838318, 1.0293447971343994, 2.7352030277252197, -0.2932180166244507, -0.3742898106575012, -0.8665105700492859, 0.5167825222015381, -2.1261608600616455, -1.0146245956420898, 1.3095906972885132, 1.0166170597076416, 1.8993717432022095, -1.5234692096710205, -2.121469497680664, 0.4100147783756256, -0.22505606710910797, 0.906699538230896, 0.45825690031051636, 0.8064504861831665, -0.9438681602478027, 1.2887083292007446, 0.4207037687301636, -0.8373292088508606, -0.952497124671936, -0.07418368756771088, -0.09387606382369995, -1.1319389343261719, 0.6214069128036499, 0.18528282642364502, 0.8725873827934265, 0.674167275428772, 0.262933611869812, -0.3043930232524872, 1.7243447303771973, 0.7035760283470154, 1.8477176427841187, 0.8688035011291504, -0.29184412956237793, 0.20407196879386902, 0.7589537501335144, -0.7250698804855347, -0.06689715385437012, 2.2484049797058105, 0.2938442528247833, -0.168071910738945, -1.0634000301361084, -0.926850438117981, 0.21591752767562866], [0.07133454084396362, 0.7148353457450867, 0.30299025774002075, 2.6983063220977783, 0.6438822746276855, -1.046799659729004, 0.25475189089775085, 0.24567288160324097, 0.2691507935523987, 0.2844610810279846, 1.0703234672546387, -1.375123381614685, 1.6282954216003418, -0.7972655296325684, 0.3128851354122162, 0.32950475811958313, 1.2040374279022217, -0.12133114039897919, 0.4905872941017151, -0.7560923099517822, 0.11129488050937653, 1.9685744047164917, -1.7743806838989258, -2.2728729248046875, -1.9120452404022217, 1.0043801069259644, 1.0721880197525024, -2.0792970657348633, -0.8097328543663025, -1.0450106859207153, 2.618394374847412, -0.9196999669075012, -0.5191524624824524, -1.0829129219055176, -1.1972941160202026, -0.872637927532196, 0.5707590579986572, -1.8738042116165161, -0.7547021508216858, -0.41233792901039124, 1.5732885599136353, -0.8595330119132996, 0.8181495070457458, 1.2229093313217163, -1.591710090637207, -1.4415929317474365, -1.5064656734466553, -1.4236130714416504, -0.5975916981697083, 1.5680660009384155, 1.8024944067001343, 0.6568261981010437, -1.1048038005828857, -0.2591405510902405, 0.8377393484115601, -0.703408420085907, 1.173264980316162, 2.5562665462493896, -0.5607303380966187, 1.2539235353469849, -0.4320758581161499, -0.04825133830308914, 0.11529505252838135, 0.7511970400810242, 0.6944982409477234, -1.596347689628601, 0.9301213026046753, -0.5063067674636841, 0.15558815002441406, 2.0846235752105713, 0.1266651451587677, 0.22328771650791168, -0.005523177795112133, -0.1498127579689026, 3.267305374145508, -1.3631607294082642, -0.46051475405693054, -0.8755132555961609, 0.8240691423416138, 0.5364123582839966, 0.4618326723575592, -1.498364806175232, -0.32984867691993713, 1.2054239511489868, -0.344414621591568, -0.2863222658634186, -1.9374562501907349, 0.7487970590591431, 0.8741854429244995, 0.9065790176391602, -0.5318374037742615, 0.9767091870307922, 1.5151416063308716, -0.7512972950935364, -0.9200834035873413, -1.2575567960739136, 3.1427676677703857, -1.7458597421646118, 1.1957439184188843, 0.2705691158771515, 1.5482487678527832, -0.47392505407333374, 0.5943010449409485, -0.6751582622528076, -0.27338269352912903, -1.0799003839492798, -0.7264427542686462, 0.0823579952120781, 0.6670200228691101, -0.22157637774944305, 0.07005307823419571, 0.1289524883031845, 0.9182084202766418, 1.2281855344772339, -1.1961523294448853, 0.6427436470985413, -0.50533527135849, -0.24590091407299042, -0.161200612783432, -0.244585782289505, 0.2601703703403473, -0.5230402946472168, 1.421515941619873, -1.1372790336608887, -0.09751351177692413, 0.24275004863739014, -1.7284033298492432, -0.29611340165138245], [0.3415869176387787, 0.4098905920982361, -1.3228322267532349, -0.3828986883163452, -0.6398298740386963, 0.98599773645401, -2.3049094676971436, -1.0593352317810059, -1.4432461261749268, -0.39486610889434814, -2.0065083503723145, -0.790986180305481, 0.5884995460510254, 0.7728180885314941, -0.11433719098567963, 0.07852449268102646, -0.5436151027679443, 1.6052054166793823, -0.3171447515487671, 1.4791266918182373, 1.3937199115753174, -1.4772061109542847, -0.23299121856689453, 0.2806151807308197, -0.6834189295768738, 0.9045206904411316, 2.0090017318725586, -0.6035058498382568, 0.6337297558784485, 0.06460379809141159, 0.24948623776435852, 1.0129286050796509, -1.0290186405181885, -0.545438289642334, 0.4826964735984802, 1.424009084701538, -0.9312601089477539, -2.3754825592041016, -1.040137529373169, 0.4650886654853821, -0.011173692531883717, -0.5758437514305115, -0.5860098004341125, -0.10959812253713608, 0.6843698620796204, -0.41180846095085144, 0.8855896592140198, 0.425368070602417, -0.6208540201187134, -0.0036329329013824463, -1.8793110847473145, 0.355538010597229, -0.32786446809768677, 0.042808420956134796, 0.8423029184341431, -0.3309938609600067, 0.5733328461647034, 0.1413014978170395, 1.9757821559906006, -0.3780826926231384, 1.388608694076538, -0.16344447433948517, -2.030566453933716, -0.11000113189220428, 0.025460122153162956, -0.7765893340110779, -1.403263807296753, -1.448992133140564, -0.6759817600250244, -1.208996057510376, 0.9754615426063538, -0.08400052785873413, 0.6872710585594177, -0.08217325806617737, -1.8924905061721802, 1.256866455078125, 0.41644009947776794, 0.18405520915985107, 0.2949616611003876, 0.9611855745315552, -0.10228385031223297, -1.7392563819885254, -0.5908181071281433, -1.7019628286361694, -0.5222721099853516, 0.08773031830787659, 0.34437263011932373, 0.3071514666080475, 0.5090596079826355, 1.3557323217391968, -0.05571447312831879, 0.26390793919563293, -1.6638439893722534, 1.3190940618515015, -0.1461753100156784, 0.06309062242507935, 0.10162671655416489, -0.6859514713287354, -1.086370825767517, -0.753956139087677, -2.6292402744293213, 2.215956211090088, -1.658315658569336, 0.3562425374984741, -1.1405614614486694, -0.298005074262619, 0.0800415500998497, 0.9069143533706665, 0.14026190340518951, -0.30947962403297424, -0.0015283634420484304, -0.3474198281764984, -2.2123591899871826, 1.641648530960083, 0.20731930434703827, -1.0525915622711182, 0.043309617787599564, -0.1451541930437088, -1.0475164651870728, 1.3578190803527832, -0.9279645681381226, -1.1706902980804443, 0.4096982479095459, -0.425187885761261, 1.9290097951889038, -0.6180810928344727, -0.3188857138156891, -0.34557148814201355], [1.4966797828674316, 0.7073333859443665, -0.5444762110710144, 0.4464268386363983, -0.766430139541626, 1.2346705198287964, 0.3758060336112976, -0.21512600779533386, -0.38094866275787354, 1.2099456787109375, 2.496716260910034, 0.06959816068410873, -2.307718276977539, -1.965876579284668, -0.2372513711452484, -0.7662196755409241, -1.2120556831359863, -0.06520646810531616, 0.2674642503261566, 0.5060209631919861, -0.8603900074958801, -0.8946056962013245, -0.0030153554398566484, -2.1358542442321777, -0.6061826348304749, -0.12509000301361084, -0.20602937042713165, -0.5997395515441895, -0.18155419826507568, -0.7368314862251282, 0.7610746026039124, -1.6856375932693481, -1.8820984363555908, -0.2988300323486328, -0.9525573253631592, 1.5839828252792358, 0.27862784266471863, -1.5327221155166626, -1.2981938123703003, -0.7468056082725525, -0.5562886595726013, -2.3339719772338867, 0.34235623478889465, -0.6391807794570923, -0.44792574644088745, -0.36671188473701477, -0.8119676113128662, -1.7196049690246582, 2.080984354019165, -1.6673343181610107, 0.06478450447320938, -0.0009864663006737828, 0.23283950984477997, -1.2802464962005615, 0.03562844917178154, -0.1632324606180191, -1.0513256788253784, -0.003958375658839941, 0.21032387018203735, -0.052201926708221436, 0.9012187719345093, -0.23182089626789093, 0.23899538815021515, 0.6196016669273376, -0.18680155277252197, 1.0550339221954346, -1.3226398229599, -0.9948657751083374, 1.5591864585876465, -0.22976624965667725, 0.69426029920578, -0.38833358883857727, -0.6811390519142151, 0.27232834696769714, 0.5953276753425598, -0.6514990925788879, -0.6657447218894958, -1.6724644899368286, 0.031348906457424164, 0.4647347033023834, 0.2267165184020996, -1.698360562324524, -0.6669340133666992, 0.700044572353363, -3.074126720428467, 0.7366824746131897, -1.3016456365585327, 0.8821328282356262, -1.4731450080871582, 0.06804698705673218, -0.39452096819877625, 0.6066170334815979, 0.034868452697992325, -1.7342917919158936, -0.39270997047424316, -1.0585957765579224, 1.0134994983673096, 1.5521057844161987, 2.8033254146575928, -0.9259083867073059, -0.5234156250953674, 0.2988153398036957, 0.11263211071491241, -1.485242247581482, -0.8884425163269043, -1.3078831434249878, -1.6670286655426025, -1.428885579109192, 0.2936629354953766, 0.7443209290504456, -0.46927350759506226, 1.5361114740371704, 1.5795270204544067, 0.20856209099292755, 0.4961397051811218, -1.8422728776931763, -0.21397764980793, -1.3786441087722778, -0.898253321647644, -0.7042690515518188, 0.9050480127334595, 0.16829226911067963, 0.37338435649871826, 1.0153156518936157, -0.4059544801712036, 1.2396979331970215, 1.2961174249649048, 1.2019543647766113], [-0.5448349714279175, -0.6538594365119934, 0.10636527836322784, -0.06705253571271896, 0.13006828725337982, 0.1301904022693634, 0.5905560851097107, -1.538641095161438, -0.6885644197463989, -1.2374238967895508, -0.3605017066001892, -1.2792719602584839, -0.5968108177185059, -1.3810325860977173, -0.6613008379936218, -1.6336978673934937, 0.04340630769729614, -0.5372775197029114, -0.14082004129886627, -0.4321914315223694, 0.05165732651948929, 1.845462441444397, -2.3429501056671143, -0.14028871059417725, -0.2839616537094116, 2.560360908508301, -0.6609848737716675, -0.08303939551115036, -1.2328057289123535, -1.5790133476257324, -1.5573227405548096, -1.167555809020996, 0.9163313508033752, 0.8388030529022217, -1.8099387884140015, 0.3712669909000397, -0.9496424794197083, -0.7950069308280945, 2.638254404067993, -1.9822415113449097, -1.688664436340332, 1.1776444911956787, 0.25015848875045776, -0.5950843691825867, -1.2127245664596558, -1.4387871026992798, 0.48468640446662903, 0.3547612130641937, -1.081121802330017, 1.852338194847107, 1.7509816884994507, -1.0914866924285889, 0.633405327796936, 2.294842004776001, 0.13764327764511108, 0.8493794202804565, 0.6315383315086365, -0.7882592678070068, -0.8836575746536255, 0.15192270278930664, 1.2484511137008667, -0.6645915508270264, -0.008799061179161072, -0.0945013016462326, 1.480391263961792, -1.7088323831558228, 1.046519160270691, -0.5420107245445251, 0.1857251226902008, 0.7044722437858582, -0.07630116492509842, -1.1792192459106445, 0.2046826183795929, 2.5013086795806885, -0.13421712815761566, -1.043571949005127, 2.4589016437530518, -0.3585955500602722, -0.7005820274353027, -1.0315923690795898, 0.362748920917511, -1.5153270959854126, -1.618046760559082, -0.34299132227897644, 0.6000509858131409, -0.7249042391777039, -1.4334992170333862, 1.0573229789733887, -2.1432077884674072, 0.1111038476228714, -0.16008202731609344, 0.37803032994270325, -2.201932907104492, -0.45881912112236023, 0.9070983529090881, -1.336633324623108, 1.7256592512130737, -0.26274392008781433, 0.28612688183784485, -1.0095791816711426, 0.3745553493499756, -2.2106528282165527, -0.8806321620941162, 1.1579087972640991, -1.0500630140304565, -0.6832361817359924, -0.7422600984573364, -2.3965070247650146, -0.2215854823589325, -0.5292284488677979, 0.02414042130112648, -0.3320809006690979, -0.7457497715950012, -0.5845398306846619, 1.3654457330703735, 1.0697705745697021, -0.6291271448135376, -0.23116420209407806, 0.7910046577453613, 0.7982575297355652, -0.566076934337616, 0.9856944680213928, -0.860139012336731, -0.3095357120037079, 1.2554229497909546, 0.2965185344219208, 0.6575779318809509, 0.06364902853965759], [2.4058048725128174, -1.3902108669281006, 1.8958390951156616, 0.7024545073509216, -1.126832127571106, -0.2966134548187256, 0.16796518862247467, 0.8057254552841187, -1.2157450914382935, 0.41616857051849365, -0.5362061858177185, -1.2104493379592896, 2.0989186763763428, -1.5665974617004395, 0.4160737097263336, 0.34445926547050476, 0.20322369039058685, -1.1933693885803223, 2.542703151702881, 0.4092884361743927, 0.783496618270874, 1.5177175998687744, -0.7341756820678711, -0.3852780759334564, -1.6329052448272705, 1.1247062683105469, -0.860927939414978, 0.38220351934432983, -1.0197453498840332, -1.0879089832305908, -0.33050477504730225, -0.47107481956481934, 0.9961860179901123, -0.7906244397163391, -0.8213198781013489, 0.6303896903991699, 2.7459516525268555, 0.36211860179901123, 1.417954921722412, 0.06590939313173294, -1.4077661037445068, 0.5026443004608154, -0.660822868347168, -1.1019630432128906, 0.7712182402610779, 1.036072850227356, -0.6232975721359253, 0.510631263256073, -0.0050443983636796474, -0.20200596749782562, -0.6846966743469238, -0.929283618927002, -0.26846078038215637, -0.5873982310295105, -2.10774302482605, -0.41585060954093933, 1.6665669679641724, -0.6492937207221985, 1.4543101787567139, 0.7990175485610962, -0.8193761110305786, -1.1137418746948242, 0.9213749766349792, 0.11822991818189621, 0.6037676334381104, 1.0437406301498413, 0.9473679661750793, 0.193405419588089, -0.800128698348999, -1.1532405614852905, -1.1680606603622437, -0.11689800024032593, -0.09297595173120499, -0.6178513765335083, -0.44855478405952454, -1.2734503746032715, 0.6983844637870789, -2.12036395072937, 0.4750642776489258, 0.9240777492523193, 0.16462449729442596, -0.025047635659575462, -0.07511328905820847, -1.0716187953948975, -1.1847504377365112, 0.14656691253185272, -1.1197408437728882, -1.021487832069397, -1.1088680028915405, -1.3403668403625488, 1.5056349039077759, 0.7405686378479004, -0.7428438067436218, -0.2620103359222412, 1.8450639247894287, 0.6302312612533569, 1.1793467998504639, -0.31689774990081787, 0.10501818358898163, -0.8604698777198792, -0.8335145115852356, 1.516767144203186, -0.40902698040008545, -0.3913189470767975, 0.003071303479373455, -0.023307843133807182, -0.3308752775192261, -1.250303030014038, 0.13005411624908447, 0.043299343436956406, 1.3519010543823242, -1.4915918111801147, -0.8144456744194031, -0.13715840876102448, 0.0863996222615242, -0.9866355061531067, -0.969819962978363, -0.627956211566925, -0.47606149315834045, 2.0120882987976074, 1.136020302772522, -0.16768574714660645, 1.1903784275054932, -1.2199848890304565, 0.15445569157600403, -0.009964637458324432, 0.31273287534713745, 0.11379015445709229], [0.5743690133094788, -0.045354537665843964, 0.3174956440925598, 1.347922921180725, -0.701888382434845, 0.8892487287521362, 0.18731144070625305, -1.0914065837860107, -1.8102271556854248, -0.7336317300796509, -0.30742573738098145, 0.9399821758270264, -0.3223530948162079, -1.0191888809204102, -0.49009644985198975, -1.002051830291748, -0.5197510123252869, 0.8464227318763733, 0.6514694690704346, 0.028718795627355576, -0.1320337951183319, -0.20015858113765717, -1.0932425260543823, 0.42949941754341125, -1.140347957611084, 1.092326283454895, -0.26297813653945923, 2.2337229251861572, -2.7061381340026855, -0.03744252398610115, 1.3643181324005127, 0.4631238281726837, -0.5318794846534729, -1.4167888164520264, 0.6926863193511963, 1.0129578113555908, -0.7011533379554749, 1.7876718044281006, -0.3881591260433197, -1.6379003524780273, -0.6566240787506104, 0.3041176199913025, 0.3579637110233307, -1.0022615194320679, 0.6426138281822205, -0.11353416740894318, -0.6764313578605652, -1.5862398147583008, 1.055436372756958, -0.5441084504127502, -1.1643871068954468, -0.2838454246520996, 0.8687134385108948, -0.24707850813865662, -0.5186542272567749, 0.6268989443778992, -2.2661149501800537, -0.26606783270835876, 0.7786635756492615, -1.1359314918518066, -0.07684430480003357, -0.17415273189544678, 0.4954302906990051, -0.2508939504623413, 1.1027116775512695, 1.063392996788025, -1.1537288427352905, -0.2519280016422272, 0.4326845705509186, -0.24109818041324615, -0.33888402581214905, 0.7468847632408142, -0.7909058332443237, -1.7501158714294434, 0.9219852685928345, -0.2477012425661087, -1.2721308469772339, -1.2824196815490723, 0.9231288433074951, 0.13264180719852448, 0.10092294216156006, 1.230846881866455, 2.066342830657959, -0.13771356642246246, 1.1687345504760742, 0.9764886498451233, -1.0446141958236694, -0.9222105145454407, -0.1625710427761078, 0.8162770867347717, -0.24977098405361176, 0.9643917083740234, -1.7332491874694824, 0.6804219484329224, -0.2418777197599411, -0.06191820651292801, 0.5298125147819519, 1.2783771753311157, -0.3615409731864929, -1.1703368425369263, 0.8248857259750366, -0.9159292578697205, 2.548825979232788, 0.8829517364501953, -1.2131645679473877, -1.2101619243621826, 1.8647629022598267, 0.9551164507865906, 0.6580321192741394, 1.3758351802825928, 0.7610732913017273, -0.06110682338476181, -1.3989200592041016, 1.17206871509552, -0.3394853472709656, 0.5919039249420166, 0.5630564093589783, -0.6099494099617004, -0.3315598964691162, -0.16089962422847748, -2.1949827671051025, 1.3511303663253784, 0.29037052392959595, -1.1855863332748413, 0.6192107796669006, -1.8575845956802368, -1.2856284379959106, 0.5031406879425049], [-1.2334742546081543, 0.41218847036361694, -0.26513928174972534, -0.635280191898346, -0.016397884115576744, -0.5176244378089905, 1.7953455448150635, -1.0779069662094116, -0.5776312351226807, -0.2485867589712143, 0.6485860347747803, 0.5543619394302368, 0.8095405697822571, 0.48485347628593445, 0.9142516255378723, 1.1816591024398804, 0.018114512786269188, -0.46271032094955444, -0.6023079752922058, -0.6899632811546326, 1.3270456790924072, -1.6810790300369263, 0.007703823037445545, -0.3045225143432617, 0.6271883249282837, 0.9169082641601562, 0.6089518666267395, 0.38796019554138184, -2.2606334686279297, 1.0720889568328857, -0.5759401917457581, 0.23882216215133667, 0.48921456933021545, 0.7237727046012878, -0.9121465682983398, 0.22792385518550873, -0.8747768402099609, 0.6193016171455383, -0.3732239902019501, -0.376904159784317, 0.3585066795349121, 1.1426706314086914, 0.21960538625717163, -0.18618600070476532, 0.599839985370636, 0.10225387662649155, 0.031620774418115616, 0.5985395312309265, 0.9210568070411682, -1.0328108072280884, -1.2405688762664795, 2.4402036666870117, 0.5971885323524475, 0.7264350652694702, 1.3735522031784058, -0.43213772773742676, -0.8343530297279358, 1.6096625328063965, -1.4492237567901611, 0.09721574187278748, 0.38244473934173584, -1.1542705297470093, -1.7326246500015259, 0.23070314526557922, -0.7300545573234558, 1.7175143957138062, 1.2670539617538452, -1.7768614292144775, 0.6330676078796387, -1.4930431842803955, -0.03326817974448204, -1.2743425369262695, 0.4351250231266022, -0.45321252942085266, -1.0679125785827637, -0.5029082894325256, 0.0044005075469613075, -0.17768089473247528, 0.12129126489162445, -0.8243196606636047, 0.12228765338659286, -0.7517713308334351, -1.0822579860687256, 0.16547268629074097, -0.9586005806922913, 0.19909313321113586, -0.7560180425643921, 0.9469872117042542, -0.09391510486602783, 0.6607877612113953, -0.5142204761505127, 1.7467676401138306, -0.17106731235980988, 0.2845803499221802, 0.3110663890838623, -0.33655378222465515, -0.6470866799354553, -0.18533611297607422, 0.6754282116889954, -0.9639864563941956, 0.7422367334365845, 0.6005297899246216, -1.585045576095581, 0.48649755120277405, 0.5095882415771484, -0.23137259483337402, 0.09910878539085388, 0.4383106529712677, -0.3901330828666687, 0.30734533071517944, 0.23837167024612427, -0.45034199953079224, 0.29271289706230164, -1.2862211465835571, 1.5311386585235596, 0.8141292929649353, 1.115992546081543, -0.009458374232053757, -0.3454938530921936, -1.0617296695709229, 0.32084983587265015, -1.8289170265197754, 2.1115128993988037, 0.020388778299093246, 0.4873046875, 0.8763259649276733, 0.8713140487670898, 2.0329010486602783]], [[-0.02232659049332142, 0.5492122173309326, -0.43870964646339417, 0.8698589205741882, -1.9724140167236328, -0.6881068348884583, 0.44065460562705994, 1.0206565856933594, -0.8193824291229248, 0.9520648121833801, -0.5163379311561584, 0.4479963779449463, 1.7136989831924438, -0.46340760588645935, -1.0240086317062378, -0.44610515236854553, 0.6273113489151001, 1.686420202255249, -1.0455392599105835, -0.8095199465751648, 0.4467152953147888, 0.7395512461662292, -1.5818357467651367, -0.4590337574481964, 0.2913369834423065, 2.042997360229492, -0.9043483138084412, -0.44358423352241516, 0.9129735231399536, 0.7177342176437378, 0.15446317195892334, 1.1900602579116821, 0.46441516280174255, -1.1652886867523193, -0.8952473402023315, -1.9147895574569702, -0.1295863389968872, 1.3957735300064087, -0.1467161774635315, -0.29881590604782104, 0.1925489902496338, 0.4503394663333893, -0.23656843602657318, 0.43253764510154724, -0.20351968705654144, -1.7456283569335938, 0.6311163306236267, 1.0942072868347168, 0.8366175293922424, 1.5515356063842773, -0.09819543361663818, -0.5840916037559509, 0.17768509685993195, -1.1270853281021118, 0.42934805154800415, -0.45232927799224854, -1.6147723197937012, -0.19351467490196228, -0.1913929283618927, -0.5523144006729126, 0.10291928797960281, 1.5649477243423462, -1.6419932842254639, -0.7398128509521484, -1.8020274639129639, -0.1258818358182907, -0.8143467307090759, -0.8412628769874573, 1.1364455223083496, -0.08276186138391495, 0.9486234784126282, 0.31883513927459717, -0.3714977204799652, 1.3638648986816406, -0.5845001339912415, 0.16047433018684387, -0.6468812227249146, -0.027242930606007576, -0.06925487518310547, -0.03524106368422508, 1.002297043800354, -0.0363704115152359, -1.8590689897537231, 1.0012985467910767, 0.0017429360887035728, 0.308217853307724, 0.6381757855415344, -2.6169285774230957, 1.6304210424423218, 0.6780731678009033, 1.9718588590621948, 0.3581024706363678, -0.8825951814651489, -0.8166766166687012, 1.0956008434295654, -0.27868661284446716, 1.8751530647277832, 0.06759238243103027, 1.1294212341308594, -0.4563477337360382, -1.2483094930648804, -0.8251011967658997, 0.5657841563224792, -1.6926994323730469, 1.0994805097579956, -1.1441571712493896, 1.6466877460479736, 0.5859740972518921, -1.2887197732925415, 0.4623083770275116, 0.6721899509429932, 0.15120211243629456, 0.3384908139705658, -0.32049304246902466, 0.5094918608665466, -0.3711431920528412, 0.7670607566833496, 0.4967474043369293, -1.2294062376022339, -1.6057430505752563, -0.7981876134872437, 0.056910742074251175, -0.8515138030052185, 1.7480943202972412, 0.30193355679512024, -0.7457902431488037, 1.3754632472991943, 0.485643208026886], [-1.7358664274215698, 0.9613970518112183, 0.20004653930664062, 1.1203564405441284, -0.1401224434375763, -1.1867263317108154, -0.5292438864707947, 1.6000005006790161, 1.0599347352981567, 0.71431565284729, 0.503566324710846, -1.6730622053146362, 0.16813834011554718, 1.3206868171691895, -1.0428543090820312, 0.0654694139957428, -0.43093374371528625, -1.4850995540618896, 1.3519482612609863, -1.9378045797348022, -0.30908411741256714, -0.06953292340040207, -0.08206508308649063, -1.6031763553619385, -1.1173244714736938, 0.06869403272867203, 0.9721238017082214, -0.6922109723091125, -0.09142600744962692, 0.2872047424316406, -1.9091638326644897, -1.6517441272735596, -0.8682170510292053, 1.4216806888580322, -1.0536713600158691, 0.6905356049537659, 1.506680965423584, -0.48872920870780945, 0.16721677780151367, -0.07475649565458298, -0.13277587294578552, -0.17858164012432098, -1.9126548767089844, 0.02047152817249298, 0.26799795031547546, 0.3098785877227783, -0.7923339009284973, 0.09559549391269684, -1.5289257764816284, -0.18129688501358032, -0.12907826900482178, -0.9856771230697632, 0.569858193397522, 0.2519271969795227, -1.0142396688461304, 0.2117399275302887, -0.48803696036338806, -1.1519324779510498, 0.4797273278236389, -0.8575888276100159, -0.4540443420410156, 2.91532301902771, 0.4530181586742401, -0.7044923305511475, -0.3215762972831726, 0.04879908263683319, 1.6995335817337036, 0.872649610042572, 0.22380699217319489, -0.56122887134552, 0.3617127537727356, 1.2959367036819458, 0.6485310196876526, 0.06130469590425491, -1.3389288187026978, -1.4534132480621338, -1.387777328491211, -0.1858081817626953, 1.8031890392303467, -0.5813176035881042, -0.6218902468681335, -1.3475415706634521, -1.4751827716827393, -0.9167149066925049, 0.3566172420978546, 0.3803144097328186, -0.018596511334180832, -1.086320400238037, -1.7089216709136963, -1.1346313953399658, 1.1531949043273926, 1.27532958984375, 0.32501181960105896, 0.6132850646972656, -1.0255334377288818, 0.4476202428340912, 0.551619827747345, 0.7523748278617859, 1.214919090270996, -0.6092587113380432, -0.09021572023630142, -1.6016716957092285, -1.9227007627487183, 1.2154951095581055, 1.5411919355392456, -1.3123278617858887, -0.24867410957813263, 0.3190152645111084, 0.19636167585849762, -0.405407190322876, -1.8701461553573608, -0.4443206489086151, 0.9571977257728577, -0.8197428584098816, 0.01678316481411457, 0.15976107120513916, -0.4919643998146057, 0.8834848999977112, 0.7881222367286682, -0.5022071599960327, -0.7435289025306702, -0.7577045559883118, -0.4193103313446045, -0.590488076210022, 1.810073733329773, 0.14964495599269867, -1.3177345991134644, -0.1084037497639656], [0.36108529567718506, 0.49060818552970886, 1.673248529434204, -0.2863500416278839, 0.38653501868247986, 1.7164063453674316, -1.5051066875457764, 0.09227596968412399, -0.6782965660095215, -0.9652935862541199, 0.5203744769096375, -0.7855282425880432, -0.8267510533332825, 0.1152765154838562, -1.2737101316452026, 2.3772969245910645, -0.49476301670074463, 0.9620822668075562, 1.045668601989746, -0.45049792528152466, 1.4263200759887695, -1.7328717708587646, 1.8884859085083008, 0.4238995909690857, -0.6036590933799744, -0.5928041934967041, -0.9689135551452637, -0.6534660458564758, 0.2974030077457428, -0.7835081815719604, 0.08263610303401947, 0.18747319281101227, 0.5306863784790039, -0.468578577041626, 0.2658470869064331, -0.9600158333778381, -1.3261915445327759, -0.5470330715179443, -0.08230407536029816, -1.41907799243927, -0.3947370946407318, 0.5698884129524231, 0.8574933409690857, 2.0856053829193115, -0.7050825953483582, -1.7092159986495972, 1.4847948551177979, -1.146223783493042, -0.7023658156394958, -0.026363801211118698, 1.2040619850158691, 1.3359256982803345, 1.1168479919433594, 0.2275322824716568, -1.487015724182129, -0.5269877314567566, 0.30397456884384155, 0.3139207363128662, -0.40222084522247314, 2.113231658935547, 0.19651848077774048, -0.13237743079662323, 0.4983868896961212, -0.1736314296722412, 0.5942854881286621, -0.38380393385887146, -0.7254018187522888, -0.13706117868423462, -1.804659366607666, -0.44980454444885254, 0.6575565338134766, 0.0649174377322197, 1.3752775192260742, 1.5122277736663818, 0.9859304428100586, 0.5321592092514038, 0.7797554135322571, -0.3663473427295685, 1.7288569211959839, 1.0535062551498413, 0.18194067478179932, 0.5397130250930786, 0.8983454704284668, 1.2475714683532715, -0.14193890988826752, 0.36595410108566284, 1.748548150062561, -2.0022120475769043, 0.25483208894729614, 0.24585427343845367, 0.35061535239219666, -1.1719890832901, -0.35524508357048035, 0.6058409810066223, -1.1765722036361694, -0.12222246825695038, -0.745930016040802, 0.8036309480667114, 0.14264826476573944, 0.8878483176231384, 2.4365921020507812, -0.8330562710762024, 0.38316094875335693, -1.7218798398971558, 0.1117083728313446, 1.3347587585449219, -0.4001356065273285, 0.08779699355363846, 0.223411425948143, 0.7560117244720459, -0.29585474729537964, -0.8203819990158081, -1.5100669860839844, -0.540627121925354, -1.663906455039978, 0.9813316464424133, 2.4901671409606934, -0.27433422207832336, 0.8810932636260986, -0.7397192120552063, 0.14644484221935272, -1.4612098932266235, -0.9440596699714661, -0.7007239460945129, -1.6931567192077637, -1.367805004119873, 0.4048422873020172, -0.36347752809524536], [-0.4131752848625183, 1.149857759475708, 0.6150870323181152, -0.41513592004776, -0.35877999663352966, 0.4958529472351074, 0.3803645968437195, -0.434680312871933, -2.1030261516571045, 0.16144774854183197, -0.2322196364402771, -0.08434635400772095, 1.1518265008926392, 0.39064154028892517, -0.2074493020772934, -0.29121968150138855, 0.2435038536787033, 0.4766187369823456, 0.45489975810050964, -0.3780326843261719, -0.6195741891860962, -0.6181633472442627, 1.4620232582092285, -1.0890475511550903, -0.5676579475402832, -0.8583394885063171, -0.15087412297725677, 0.7098535299301147, 1.3849546909332275, 0.9939587712287903, 0.5934526920318604, 1.090602159500122, -1.3981060981750488, -0.3953050374984741, -1.471052646636963, -0.560102105140686, 1.847769021987915, 2.6869375705718994, 0.13261330127716064, 1.2771668434143066, -0.7477888464927673, 0.6802242398262024, 1.1894397735595703, 0.4827285706996918, 2.4528093338012695, 0.7839869260787964, -0.8460619449615479, -2.023075819015503, 0.2447407841682434, -1.5974504947662354, -2.061551332473755, -1.012117862701416, 0.4408528804779053, 0.4258333444595337, 0.604849100112915, 0.19716301560401917, -1.2466721534729004, 2.3821609020233154, -0.26516154408454895, -0.9373996257781982, 0.9189923405647278, 1.9844634532928467, -0.27306246757507324, -1.1262305974960327, 0.5651792883872986, -1.5009175539016724, 0.06044352054595947, -1.4720538854599, -2.0030126571655273, -0.9687616229057312, 1.027001976966858, 1.8902260065078735, -0.04692119359970093, 1.054783821105957, -0.03792066499590874, -0.5090529322624207, 0.49077877402305603, -0.22986476123332977, -0.15477712452411652, -0.7282716035842896, 2.38210129737854, -0.05638781189918518, -0.7760895490646362, 0.4110625088214874, 0.3621416389942169, 0.8660764694213867, -2.0390372276306152, -0.7653892040252686, -0.008614715188741684, -0.0850643739104271, -0.32091024518013, -0.6636326909065247, -1.469576358795166, -2.0240895748138428, 0.08999020606279373, -0.8509358763694763, -0.2465633749961853, 1.0784636735916138, -0.503360390663147, -1.9154765605926514, -1.6729971170425415, -1.176041841506958, -1.4668861627578735, 1.088753342628479, -1.0974931716918945, 1.362883448600769, 0.45352450013160706, 0.5560451149940491, 4.281327724456787, 0.9223714470863342, -0.7717582583427429, -0.7630108594894409, 0.7966611981391907, 1.5838226079940796, 0.015580396167933941, -0.22153392434120178, -0.3614543676376343, 0.7364996671676636, 0.08006581664085388, 1.0173002481460571, 0.9560282826423645, 2.563400983810425, -0.2785210609436035, -0.9117327332496643, 0.46172258257865906, -0.0966092050075531, 0.3352668881416321, 0.09909922629594803], [0.5711860656738281, -1.7721078395843506, -0.7031840085983276, 1.2098642587661743, 0.17450714111328125, -0.44580787420272827, 0.45195427536964417, 0.0002872189215850085, 1.6456149816513062, -1.9784095287322998, 1.3087162971496582, 0.0028875565622001886, 0.29116740822792053, -0.6672147512435913, 0.4104992151260376, -1.8803534507751465, 0.27370432019233704, -0.339764803647995, -1.4362326860427856, -0.8520123362541199, 1.1737074851989746, -0.3708323538303375, -0.8292973637580872, 2.16629695892334, 0.3318958282470703, 0.9130294322967529, 0.6483401656150818, 0.6331278085708618, 0.6278990507125854, -0.22972813248634338, -0.37336623668670654, -1.9514042139053345, 1.175121545791626, -0.3971931040287018, -0.8093805313110352, -1.8339735269546509, -0.6917242407798767, -0.23870491981506348, 1.7139009237289429, -1.4713026285171509, 1.0772768259048462, -0.22456026077270508, -0.5165987610816956, -0.4101913273334503, 0.1944994181394577, -1.5594438314437866, -0.06231915205717087, -0.6612727046012878, 0.6168737411499023, 0.48854026198387146, 1.1140456199645996, 0.8122462034225464, 0.170523539185524, 1.0139366388320923, 0.29426735639572144, -1.3327751159667969, -1.165544033050537, -1.478564977645874, 0.2552398443222046, 1.3302611112594604, -0.3121016323566437, -1.7823357582092285, 0.18119069933891296, 1.3441096544265747, 0.09647859632968903, 0.45221906900405884, -0.4436382055282593, -1.2821873426437378, 0.07430703938007355, 0.34887179732322693, -0.6124049425125122, 0.04475967958569527, 0.6926446557044983, 0.09230564534664154, 1.024398922920227, 1.2297877073287964, 1.6451804637908936, -1.0693705081939697, 2.6097066402435303, 0.19108489155769348, 0.26362964510917664, 0.4125956892967224, -2.272414445877075, -1.5350697040557861, -0.8081147074699402, 0.017084501683712006, -1.0184516906738281, 0.7974735498428345, -1.957371473312378, 0.6163715124130249, -1.1640214920043945, 0.010232930071651936, -0.49252644181251526, -0.5408584475517273, 0.11187315732240677, -0.9686604142189026, 0.6840080618858337, -0.4419757127761841, 0.4405241906642914, 1.0256297588348389, -0.2284557968378067, 0.5371611714363098, 1.039677381515503, -0.7839885354042053, 0.570958137512207, 0.2769690454006195, 1.2703746557235718, 0.4154297709465027, -0.12052372097969055, 1.7206835746765137, -0.881895899772644, 0.4878285229206085, 2.1084277629852295, -0.9183589816093445, -1.269431471824646, -0.48702022433280945, 0.5763892531394958, -0.20751290023326874, -1.2112212181091309, 0.7766714096069336, 0.03925824537873268, -0.17255601286888123, -0.3901045620441437, 0.6546862125396729, -1.1442482471466064, 0.3106462359428406, -1.205079197883606, -0.6410843133926392], [0.1200333684682846, -0.6118667721748352, 0.08373622596263885, 1.0542627573013306, -0.8648779988288879, 0.2575826048851013, -1.6113141775131226, -0.24820546805858612, 0.5047421455383301, -1.6492441892623901, -1.7538777589797974, -0.1086413562297821, 0.01640067994594574, 1.5819658041000366, -0.7190134525299072, -0.21362215280532837, -1.4741809368133545, 0.32209548354148865, 0.09675177931785583, -0.6193909049034119, -2.1475744247436523, -1.453262209892273, -0.44714540243148804, 1.119722843170166, -0.12588749825954437, -0.08061880618333817, 0.40183842182159424, -0.7536364197731018, -0.3830912411212921, -1.1198238134384155, 0.5393616557121277, -1.9687491655349731, -1.1116642951965332, -0.12495908886194229, -0.3734111785888672, 1.004995584487915, -1.135738492012024, -0.585446834564209, -0.44423624873161316, -0.8700150847434998, 1.5265752077102661, -0.36361873149871826, 0.7603497505187988, -1.6978046894073486, 1.3385465145111084, 0.5545929670333862, 0.32534059882164, -1.4695281982421875, 0.5900099873542786, 0.18440808355808258, 1.0484187602996826, 2.4318790435791016, 0.34530115127563477, 1.0912786722183228, -1.0575023889541626, -1.62277352809906, 0.7303787469863892, 0.4597598612308502, 0.0843803882598877, 0.5370280146598816, 1.5622907876968384, -0.7496780753135681, 0.20086054503917694, -0.9663623571395874, -0.4174153804779053, -0.9478012323379517, -0.4329679608345032, 0.47176027297973633, 0.25327372550964355, -0.11184454709291458, -0.28293293714523315, 0.566745936870575, 0.2750076651573181, 0.17239049077033997, 0.7101423740386963, 2.2508347034454346, 0.501803457736969, -1.047392725944519, -0.038662537932395935, 0.3871432840824127, -0.7568838596343994, -1.0062235593795776, 1.1552467346191406, -0.971410870552063, -1.4096431732177734, -0.23661920428276062, -0.3694871962070465, -2.2273552417755127, -0.2586001455783844, 0.009799408726394176, 0.3579701781272888, -0.8885435461997986, -0.7345865368843079, -0.5291603207588196, -1.0094283819198608, 0.6434112787246704, -0.9286400079727173, 0.785271167755127, -0.818156898021698, -1.125831127166748, 1.218975305557251, -0.39321690797805786, -1.6684712171554565, 0.10261324793100357, 1.3118281364440918, 1.3106364011764526, -0.32777532935142517, -0.928784966468811, -0.0379442498087883, 0.4560682773590088, 0.40618354082107544, -0.6391202211380005, -0.8827879428863525, -0.8432422280311584, 0.49994033575057983, 1.0537104606628418, 1.708987832069397, -0.42362216114997864, 0.18065804243087769, 0.2870109975337982, -0.8811005353927612, 0.3806893527507782, 2.0902774333953857, -0.8560421466827393, -1.4638919830322266, 1.3688504695892334, 0.5940128564834595, 1.049311637878418], [-2.1618452072143555, 0.2943463623523712, 0.5469056963920593, 1.4165723323822021, 2.25289249420166, -0.6771288514137268, -1.1225571632385254, -1.2316415309906006, -1.2754539251327515, -1.163430094718933, 0.12444424629211426, 0.5637683272361755, 2.562549591064453, -1.991459846496582, -0.1024840772151947, 0.2703397870063782, 0.10454101860523224, -1.1472219228744507, 1.4522989988327026, -1.1045093536376953, 0.4675340950489044, 0.26090994477272034, -0.8047597408294678, 1.0643342733383179, -0.30578333139419556, -1.8577090501785278, 0.4661499261856079, -0.4990233778953552, 1.4631210565567017, 0.18526311218738556, 0.34530240297317505, -1.28499174118042, 1.1273730993270874, -0.008060397580265999, -2.410706043243408, 0.34797829389572144, 0.8127980828285217, -0.1692570447921753, -0.03642743080854416, -0.938896119594574, 1.1527206897735596, -0.20835396647453308, 0.1323864459991455, 0.46551674604415894, 2.4236042499542236, 0.10395902395248413, 0.6022834777832031, 0.49573346972465515, -0.4882145822048187, -0.24676913022994995, 0.31237074732780457, -1.0560663938522339, -0.5044716596603394, 0.19214949011802673, 0.6985818147659302, -1.0409307479858398, -2.3439388275146484, 1.0537959337234497, -0.6869436502456665, -0.43975844979286194, -0.3158901631832123, -1.841516137123108, 1.3318660259246826, -0.6660462617874146, -0.7050510048866272, -1.1141310930252075, 0.1027405634522438, 0.1871212273836136, -0.09646707028150558, -0.7538343667984009, -0.10414943844079971, 0.3596024215221405, -0.4283563196659088, 1.6093673706054688, -1.897183895111084, 0.824104905128479, -1.4828541278839111, -1.4009534120559692, -0.8529163002967834, -0.2226434350013733, 1.7469931840896606, 1.2868642807006836, -0.5545173287391663, -0.27211084961891174, 0.9811763167381287, -1.090315341949463, -0.7016432881355286, 0.7659969329833984, -0.7251871824264526, -1.4155765771865845, 1.343217134475708, -0.9247501492500305, -0.2188810408115387, -0.9373207688331604, 0.5839827656745911, 1.2953914403915405, -0.4308485984802246, 1.6894440650939941, -0.8341363072395325, -1.1229513883590698, -0.497365266084671, 0.6122172474861145, -0.14670224487781525, 0.154075026512146, 0.6280229687690735, -0.7128350138664246, -1.7929821014404297, -0.10835538059473038, -0.04837416857481003, 0.4859878122806549, 0.6064451932907104, 0.1745833158493042, 1.269078254699707, -1.051708459854126, -0.5035620331764221, -1.0148835182189941, -0.6509049534797668, 1.1250778436660767, 0.2276574671268463, 0.5869459509849548, 0.3268173635005951, 0.6123296022415161, 1.0660426616668701, 0.5592012405395508, 0.1836043894290924, -0.12428824603557587, 1.1998252868652344, -0.4504694640636444], [-0.37267187237739563, 1.8393280506134033, 0.3736141324043274, 0.7073833346366882, -1.7479914426803589, 0.2859545946121216, -0.38442516326904297, 0.005071087274700403, -0.6294834613800049, -1.5497397184371948, 0.8113913536071777, -0.7326053380966187, 0.04056797921657562, 0.22084252536296844, 1.1022601127624512, 0.17364241182804108, 0.9053553938865662, 0.5279042720794678, -0.5011485815048218, -1.3098465204238892, 0.7023482918739319, -0.6114794015884399, -0.08511195331811905, 0.3985387980937958, -0.6207908391952515, -2.504018783569336, 1.2381229400634766, -0.39424338936805725, -1.0470184087753296, 2.0567784309387207, -0.698154628276825, 0.903168797492981, -0.7103316187858582, -0.09143266826868057, -0.4890514016151428, -1.747763991355896, 0.09875009208917618, 0.3064643442630768, -0.2487364411354065, 0.44678783416748047, -1.1616699695587158, -0.02032547816634178, -0.12452185153961182, 1.315738558769226, 0.3589601516723633, -0.11187997460365295, 0.9147673845291138, -0.26658934354782104, -0.2666361331939697, -1.2768949270248413, -0.6114610433578491, 1.3537579774856567, 0.8036624789237976, 0.1612207442522049, -0.9639586806297302, -0.32266584038734436, 1.5867695808410645, 1.096129298210144, -0.2000686228275299, 0.5218972563743591, 0.9648618102073669, 0.7234771847724915, 0.36908215284347534, -1.2264404296875, 0.5821656584739685, 1.0529627799987793, -0.5551102161407471, -0.9850900173187256, 0.8447071313858032, 0.24604861438274384, -1.2588670253753662, -0.733513593673706, -0.560303807258606, -0.1551855355501175, 0.11106453090906143, -0.9069989323616028, 0.7797877192497253, 1.9004536867141724, -1.7101545333862305, 0.44796159863471985, 0.3287681043148041, 0.11739297211170197, -1.635632038116455, 0.22119610011577606, 0.3673020899295807, -0.04214920848608017, -0.18923884630203247, -0.033568982034921646, 0.3552871644496918, 1.0781453847885132, -0.08764390647411346, -0.7675920724868774, -0.609124481678009, 1.5375452041625977, 1.787524700164795, 0.21896880865097046, -0.5908558964729309, -0.0018691284349188209, -1.144275426864624, -0.48854973912239075, 0.10210905969142914, 1.4664252996444702, -0.6482673287391663, 0.8786709308624268, 1.0696470737457275, 0.9310802817344666, -1.3519822359085083, -0.6445575952529907, 0.5659629702568054, 0.7638929486274719, -0.36262860894203186, 0.09205850213766098, -0.523165762424469, -0.26058629155158997, 0.023233095183968544, -0.6541089415550232, 1.9593946933746338, 0.051631368696689606, -0.21110598742961884, 0.5600379705429077, -0.1588316559791565, -1.0568493604660034, -0.022205233573913574, -0.9207631945610046, -0.5931490659713745, 0.43010690808296204, -0.5460665822029114, 0.7342853546142578]], [[0.46168383955955505, -0.30537188053131104, -0.2754516899585724, -1.224083662033081, -1.9890539646148682, -0.5726816654205322, -0.9965290427207947, 0.39753788709640503, 0.10110440105199814, 1.0294184684753418, -1.3157851696014404, 1.7964717149734497, -0.13103362917900085, -0.1837068349123001, -1.4867911338806152, 1.783165454864502, 0.16251921653747559, 0.19953247904777527, -0.1156148761510849, 0.6272085905075073, 0.8698862195014954, -1.435655951499939, -2.010789155960083, 0.03307357802987099, -0.9915995001792908, 1.7794039249420166, 0.48784253001213074, -0.13488256931304932, 0.7265862226486206, 0.9411681294441223, 0.004060063511133194, 1.3989776372909546, 0.009084327146410942, -0.056334201246500015, -0.6017005443572998, 0.038076672703027725, 0.7697473168373108, -0.0872168317437172, -0.2709527611732483, -2.0055296421051025, 0.18883250653743744, -1.1298984289169312, 0.8920020461082458, 1.3919329643249512, 0.048595067113637924, 0.2309088110923767, 0.11525668948888779, -0.1879207342863083, 1.0632528066635132, 0.4122997224330902, 0.8501003980636597, 0.924876868724823, 1.0407192707061768, -3.142666816711426, -0.5162444114685059, -0.12864291667938232, 0.7980791330337524, 0.6382358074188232, -1.170776128768921, -0.9917412996292114, -0.4116307199001312, 0.14336980879306793, 0.39102616906166077, 0.1772959679365158, 0.9159114360809326, -0.8903151154518127, -1.0796746015548706, -0.6160085797309875, -0.7205215692520142, -0.17468507587909698, 0.2980307638645172, -1.3375869989395142, -1.1540772914886475, 0.728628396987915, 0.794611394405365, -0.5621936917304993, -0.9882407784461975, 0.534228503704071, -0.5202963352203369, -0.011771297082304955, 0.12000752985477448, -0.9438003301620483, -1.014075517654419, -0.4878115653991699, 0.5253517627716064, 0.9743061065673828, -0.4609658420085907, 2.498584032058716, 0.41705021262168884, -1.1283737421035767, 0.6155629754066467, -1.5193818807601929, 0.5165935158729553, -0.9670253992080688, 0.562608540058136, -0.5208752155303955, 0.5875436067581177, -0.7951277494430542, 1.387553334236145, 0.893072247505188, -0.01990421861410141, -0.1699533611536026, 0.4688454866409302, -0.3061636984348297, 2.04117751121521, -0.11754161864519119, -0.16321370005607605, 1.0240414142608643, 1.4953211545944214, -1.94973886013031, -0.2717195153236389, -2.0571603775024414, -0.270232230424881, 0.11800985038280487, -0.5964910387992859, -1.1893516778945923, -0.508133053779602, 0.7958914041519165, 1.2568109035491943, -0.7959411144256592, -0.06935907900333405, 0.7367619872093201, 0.113163061439991, 0.21933938562870026, -0.6279654502868652, -0.06576847285032272, 0.8391033411026001, 1.316625714302063], [-0.9258018136024475, 0.7453300952911377, -0.7479433417320251, -0.1511736661195755, -0.8888847231864929, 0.0835358053445816, 0.04968646541237831, 0.35568711161613464, -0.9645388126373291, -0.9416454434394836, 0.1705065369606018, -0.6453060507774353, 0.9772001504898071, 0.15997518599033356, 0.1255301833152771, -1.8589000701904297, -0.10001667588949203, 0.5168284773826599, -2.325120449066162, 0.08715158700942993, 0.2888321280479431, -0.23709018528461456, -1.464246392250061, -0.7452138662338257, 0.06964156776666641, 0.8299271464347839, -0.37824156880378723, 1.2474136352539062, -2.3878324031829834, -1.7314212322235107, -0.605253279209137, -1.21511709690094, -0.9127391576766968, -2.2843737602233887, -0.9781044125556946, 1.1502870321273804, 1.5422897338867188, -1.94627046585083, -0.27365127205848694, -0.7834264636039734, -1.2886993885040283, 3.652784824371338, 1.8363219499588013, 0.36924171447753906, 0.5972329378128052, 1.5861916542053223, -0.13794444501399994, 0.3046022653579712, -0.48233726620674133, -1.3042018413543701, 1.0623674392700195, 0.6589610576629639, 0.9601899981498718, -1.6640146970748901, -0.20913828909397125, 0.21348024904727936, -1.2470779418945312, -0.40533772110939026, -0.1722433865070343, -0.8712415099143982, -1.2622829675674438, 0.9589522480964661, -1.4615652561187744, 0.06829369813203812, -1.7633346319198608, -0.3546220362186432, 0.06314671784639359, -0.010435514152050018, 1.44383704662323, 0.30809485912323, -0.878699541091919, -2.4532217979431152, -1.15933096408844, -0.226925328373909, 0.8298043608665466, 0.12691478431224823, -0.22029195725917816, 0.5013371706008911, 0.6902851462364197, -1.2222583293914795, 0.49464914202690125, 0.1891889125108719, 3.214515209197998, -0.8369815945625305, 0.1361602395772934, 0.4843536615371704, 0.005578381009399891, -1.1416282653808594, -1.8459529876708984, -2.2103805541992188, 1.5320971012115479, -0.2651122212409973, 1.3941760063171387, -0.7014652490615845, 1.277225136756897, -1.672487735748291, 0.08014381676912308, -0.5036567449569702, -0.8954418897628784, -0.4108841121196747, -1.3719093799591064, -0.2951678931713104, 0.20530813932418823, 0.23351995646953583, -0.37169384956359863, 0.6958168148994446, -1.0963447093963623, -0.3521628677845001, 0.0571015402674675, 0.9459565877914429, -0.667320191860199, -0.7150914669036865, -0.3362846374511719, -1.7354401350021362, 0.7449601292610168, 0.8260166645050049, 0.8950182795524597, 0.12075911462306976, 0.6555850505828857, 2.3638460636138916, 1.2178767919540405, 0.006517596542835236, 0.11981307715177536, 0.9935739040374756, 0.5003350377082825, 1.2661099433898926, -1.2940841913223267, 0.7891700267791748], [-1.083466649055481, 0.41245076060295105, -0.30474862456321716, 0.08486954867839813, -0.6401517987251282, 0.6023409962654114, 1.7551424503326416, 0.44725897908210754, -0.7762457728385925, -1.023154854774475, 0.37569963932037354, 1.3335857391357422, 0.18237555027008057, 0.8630683422088623, -1.098405122756958, 0.8463390469551086, -0.16172859072685242, -0.5229173302650452, -0.9694331288337708, -1.0614374876022339, 0.6931996941566467, 0.1370846927165985, -0.27625763416290283, -0.27680066227912903, -0.46820858120918274, 0.6938672661781311, 0.47901690006256104, 0.4609850347042084, 1.543866515159607, -1.2829238176345825, 1.172114372253418, 0.005610391031950712, 0.1397198736667633, -1.9022107124328613, 1.2661638259887695, -1.143760085105896, -0.7966916561126709, -0.946739137172699, 0.06136200577020645, -0.7728052735328674, 0.6556852459907532, -0.6032726764678955, 0.8003679513931274, -0.29514241218566895, 0.48021793365478516, 0.38096484541893005, 0.33178406953811646, 0.37361669540405273, -0.2795178294181824, -0.36682751774787903, -0.48571011424064636, 1.494938850402832, -1.5778923034667969, 0.24528682231903076, -0.20849065482616425, -1.507204532623291, -0.7838833332061768, -1.476050615310669, 0.4491197466850281, 0.5142097473144531, -1.3036860227584839, -1.5491766929626465, -0.5521824359893799, -0.38058745861053467, -0.5702511668205261, -0.7317973971366882, -1.2059048414230347, 0.4024665951728821, 0.37127768993377686, 0.67927485704422, -0.7502949237823486, 1.1201497316360474, 0.09088165313005447, -1.5044832229614258, 2.2387259006500244, 0.046747881919145584, 0.31423380970954895, 1.136379599571228, -1.0610589981079102, 0.1805715262889862, -0.26328298449516296, -0.7965911626815796, 1.5842137336730957, 1.9697755575180054, 0.7795166373252869, -0.6216588616371155, -0.7145920991897583, -0.9586138725280762, -1.0141122341156006, 1.6089808940887451, 2.0019280910491943, -0.39478233456611633, 2.06284236907959, -2.0082757472991943, -1.1909111738204956, 0.3614574372768402, -0.9987539052963257, 0.9056900143623352, 0.5387725234031677, 0.026587212458252907, -0.035902973264455795, -1.6085233688354492, 1.604175329208374, -0.7770441770553589, 1.657763123512268, -2.0814247131347656, 0.45214930176734924, -0.36292359232902527, -0.7451586723327637, -0.4245636463165283, -0.5613599419593811, -2.1125266551971436, -0.45445460081100464, 0.9569663405418396, 0.13739833235740662, 0.2667091488838196, -0.32099586725234985, -0.7644869685173035, -0.5221232771873474, -1.194640874862671, 0.14676791429519653, 0.1845475435256958, 1.0643980503082275, -0.19979362189769745, 0.26775792241096497, -1.8104541301727295, -0.6897791028022766, 0.1790628433227539], [0.902062177658081, 1.1180663108825684, -0.5635412931442261, 0.09204874187707901, 0.028015341609716415, 0.21295997500419617, -0.39112669229507446, 0.941992998123169, -0.5093554258346558, -0.40142980217933655, -0.2166198492050171, 0.2841435670852661, 0.6043798923492432, 1.4133124351501465, 0.2973203659057617, 1.6884709596633911, -1.013494610786438, -0.253693550825119, -1.3128962516784668, -1.6159261465072632, -0.6496108770370483, -0.6973762512207031, -0.20087575912475586, -0.14024749398231506, -0.424021452665329, -1.9596240520477295, -0.16316856443881989, 0.5600936412811279, 0.19636943936347961, -0.46986323595046997, -0.6233101487159729, -0.7654959559440613, -0.5636140704154968, -0.21877603232860565, -1.2810221910476685, 0.33476385474205017, 0.9689939618110657, -0.40640565752983093, 0.9247424602508545, -2.384192943572998, -0.9818882942199707, 0.8178919553756714, -0.3951044976711273, 0.1393093764781952, -0.24950413405895233, 0.2880459427833557, -1.0263320207595825, 0.6533238291740417, -1.3759479522705078, 0.797831118106842, 0.7408795356750488, 0.39077794551849365, -0.31545382738113403, -0.6051350831985474, -0.05480601266026497, 0.5422443151473999, 1.6613761186599731, -0.6383312940597534, 0.4519702196121216, -0.05824485048651695, -0.4333164691925049, -0.48517856001853943, 0.4341345429420471, -1.1917524337768555, -0.7748887538909912, -0.6490197777748108, 1.0936756134033203, -1.2035678625106812, 1.518420934677124, 0.9012187123298645, -0.3745622932910919, 0.5961492657661438, -0.21467822790145874, 1.0793352127075195, 1.1577234268188477, -1.1488454341888428, 1.5175596475601196, -0.4964780807495117, 0.3117784857749939, -1.5624933242797852, -0.75553297996521, 0.35165664553642273, -0.957210898399353, -0.5085762143135071, -0.5149138569831848, 1.517225742340088, 2.7367255687713623, -0.9329893589019775, 0.2648203372955322, -1.0009351968765259, -2.186051845550537, -0.48564648628234863, 0.02720114402472973, -0.37736645340919495, -0.37229737639427185, 0.8066195249557495, -1.1785924434661865, -0.6137113571166992, -0.9677262902259827, -0.9082260727882385, 0.9475095272064209, -1.9403233528137207, 0.10285356640815735, 0.03520740196108818, 0.45327824354171753, -2.3853023052215576, -1.630932092666626, 1.12734854221344, -2.3434102535247803, -0.1375761479139328, 0.464125394821167, -2.5417115688323975, 0.4301947057247162, 1.0248513221740723, -2.122265100479126, -1.4702143669128418, 1.194297432899475, -0.9389809966087341, 0.8018950819969177, -0.3240561783313751, -2.0579421520233154, -0.9603367447853088, 1.1335171461105347, 1.5070273876190186, -1.0619118213653564, 1.0732442140579224, 0.5316171050071716, -0.18741166591644287], [0.9404513835906982, -0.6696039438247681, 0.9144638180732727, 0.18118242919445038, 1.0554661750793457, -1.6418805122375488, -0.002680805278941989, -0.12439475208520889, 1.080740213394165, 2.5475971698760986, 0.7883082628250122, 1.1349847316741943, 0.2806272506713867, -0.10223424434661865, 0.7364910840988159, 1.1261827945709229, 1.803293228149414, -2.0283334255218506, -0.7161402106285095, -0.0035189089830964804, 1.2448093891143799, -0.21615706384181976, -1.6234982013702393, -1.5583558082580566, -1.933296799659729, -1.9575581550598145, 0.6826256513595581, 1.122220754623413, -0.5354470610618591, -0.529900848865509, -1.4400957822799683, -0.8890421986579895, -0.14221671223640442, 0.14864511787891388, 0.5840739011764526, 1.8109923601150513, 1.0220539569854736, -2.1043848991394043, -0.7077109217643738, -0.5559590458869934, -0.5291451811790466, 0.5455774068832397, 1.358083724975586, -0.1848398894071579, 1.014345407485962, -0.4092385172843933, -0.6812956929206848, -0.7189990878105164, 1.2259327173233032, 0.24441614747047424, -1.117154598236084, 0.9449842572212219, -0.47300201654434204, -0.011032244190573692, 0.07913072407245636, 0.1677297055721283, 0.4023098349571228, -1.1581276655197144, -0.180304616689682, 0.8758583664894104, 1.879490613937378, -0.8536171317100525, -0.02121628075838089, -1.7288950681686401, -0.45525243878364563, 1.4557515382766724, -0.4902852475643158, 0.18176141381263733, -0.0593746155500412, 0.9283919334411621, 0.009636392816901207, 0.7406389117240906, -0.17619523406028748, 0.2014879286289215, -0.04572094604372978, -0.9592661261558533, 0.6225705146789551, -0.9916889667510986, -0.9920145869255066, -0.896285891532898, -0.11859698593616486, -0.6228218674659729, 0.200749471783638, 0.5398014783859253, 0.28174644708633423, 0.8388328552246094, -0.06596728414297104, 0.8047500848770142, -0.9371095299720764, -0.42496636509895325, -0.424410343170166, -0.17260468006134033, 0.3747439384460449, 1.043106198310852, 0.7694196105003357, 2.2297441959381104, 0.4686475396156311, -0.1558614820241928, 0.3701018989086151, 0.850459635257721, 0.4533098042011261, -0.4773571789264679, 0.8376898169517517, 0.47241368889808655, -0.8189998865127563, 1.1459505558013916, -0.9794555902481079, -0.4965537190437317, -1.4931159019470215, 1.0644865036010742, 0.04253407567739487, -2.186537742614746, 0.8028876185417175, 0.6516212821006775, 0.550905704498291, 0.8700743317604065, 1.7488455772399902, -0.9258606433868408, -1.0030068159103394, 0.25549620389938354, -0.8907393217086792, 2.5177812576293945, -0.857945442199707, 1.0380803346633911, -0.23796327412128448, -1.422847032546997, -0.6114303469657898, 0.246027410030365], [0.41961127519607544, -0.020550601184368134, -1.1832693815231323, 0.9178754687309265, -0.04627080261707306, 0.26679331064224243, -2.014244318008423, 0.9956936240196228, 0.9372719526290894, -1.137536883354187, 0.077670618891716, 1.1085319519042969, -1.1087632179260254, -0.75975501537323, -1.946031928062439, 1.9111474752426147, 0.6634350419044495, 0.3897455632686615, 0.42097947001457214, 0.5318268537521362, 0.5048468112945557, -0.6672182679176331, 0.9850485324859619, -1.5839916467666626, 0.2709123492240906, 1.1045054197311401, 1.8704650402069092, -0.445751816034317, -1.3003273010253906, -0.10218959301710129, 0.37918123602867126, 0.9838548898696899, -0.7466245889663696, 0.8015487194061279, 0.7279331684112549, 1.0779954195022583, 1.178902506828308, -0.6144586205482483, -0.02154441736638546, -1.2621536254882812, 1.4384735822677612, 2.1037089824676514, 1.3634099960327148, -0.7375184893608093, -1.5353171825408936, 0.21421581506729126, 0.5487058758735657, 0.5352146029472351, 1.6426821947097778, 1.253835916519165, -1.6104363203048706, -0.8868870735168457, -1.0986546277999878, 0.1650010049343109, 0.40821573138237, -0.9821721911430359, -1.0796884298324585, 0.837393045425415, 1.2912914752960205, -1.000857949256897, 1.1725069284439087, 0.0727815255522728, -0.8557618856430054, 0.10401702672243118, -1.6422313451766968, -0.009844549000263214, -0.18095943331718445, 0.7984799146652222, -0.17505241930484772, -1.0124412775039673, 0.6891806721687317, 0.3770461082458496, -0.9825286269187927, 0.7246379256248474, 2.890636444091797, 0.1886642873287201, -1.2245742082595825, -1.1834033727645874, 0.8208039402961731, 1.3386532068252563, -0.5304070115089417, -0.20642532408237457, 0.6373718976974487, -0.013273252174258232, -0.2811596095561981, 0.4769556522369385, 0.20329153537750244, -0.5553554892539978, -1.4470661878585815, -0.11341017484664917, 2.203173875808716, -0.3048880100250244, -1.544276237487793, 1.2724511623382568, 1.5844212770462036, 0.623268723487854, 0.16467706859111786, -1.0350531339645386, -2.3547823429107666, -0.11177239567041397, -0.1297294795513153, 0.13905566930770874, 0.4846690893173218, -0.6218752264976501, 1.1654733419418335, 0.28632691502571106, -1.447062373161316, 0.49752089381217957, 1.2994881868362427, 0.004940879065543413, -1.604668378829956, 0.3161548674106598, -1.0069079399108887, -0.5197027921676636, -0.31187382340431213, -1.1515257358551025, 0.2067655771970749, -1.6027584075927734, 1.0394022464752197, 0.7995322942733765, -0.3863550126552582, -0.9173177480697632, -0.28930315375328064, 0.7349114418029785, 0.3367517590522766, -0.926121175289154, 0.3610716760158539, 0.8632071614265442], [-1.4006812572479248, -2.5285539627075195, -0.7031235694885254, 1.5577974319458008, -0.6477392911911011, 0.12211231887340546, -0.7347192764282227, 1.2906763553619385, 0.00013320775178726763, 0.005550843197852373, -0.07960241287946701, 0.9555078744888306, -0.22995638847351074, 0.6546006202697754, 0.7089540362358093, -0.28825509548187256, 0.01551004871726036, -0.012479974888265133, -0.10112307965755463, -1.357733130455017, 0.037508632987737656, -0.6723328232765198, 0.22181247174739838, -0.5945612192153931, 0.3931737244129181, -0.1102631688117981, 1.309531569480896, -1.3831766843795776, -0.31292739510536194, 0.5218496322631836, -0.47742781043052673, -3.653031587600708, 0.1274293065071106, -1.698256015777588, 1.4958454370498657, 0.04474148526787758, -0.8375721573829651, 0.9315623641014099, 2.1161017417907715, 0.8288314342498779, -1.0368082523345947, 1.4775135517120361, -0.16792236268520355, -1.5362752676010132, 0.7989965081214905, 0.6041649580001831, -0.9614294767379761, -0.7553507089614868, 0.3291189968585968, -1.5658220052719116, -0.12738355994224548, 3.866520643234253, 0.3258928656578064, 1.069357991218567, 0.8630348443984985, -0.5257994532585144, -0.550861656665802, -0.3240438401699066, -0.35236889123916626, -0.3056611120700836, -0.5459573864936829, 1.319994330406189, 0.7260675430297852, 0.8871457576751709, -0.05375175550580025, -0.6200947165489197, 0.5639743804931641, -1.098748803138733, 0.9620282053947449, 0.3087821304798126, 0.9724360108375549, -0.2804926633834839, 0.022380178794264793, 0.23186303675174713, -0.33081814646720886, 0.40929779410362244, 0.04763432964682579, 0.16560973227024078, 0.717654287815094, -0.6242541670799255, 1.8928093910217285, -1.246741771697998, 0.6297621726989746, -0.5517261624336243, -0.4531334340572357, 1.443579912185669, 0.1755598783493042, -0.8298792839050293, 0.2822388708591461, 0.4075991213321686, 0.4907941520214081, 1.5307395458221436, -0.07653732597827911, 1.1098381280899048, -0.07006163150072098, -1.1502337455749512, 2.3690271377563477, 2.041062593460083, 1.2236289978027344, -0.7401612401008606, 1.0700876712799072, -0.4555218815803528, -0.0015093308174982667, -0.9068090319633484, 0.8516572713851929, 0.9075549840927124, -0.6121871471405029, -0.24279659986495972, -0.04771443456411362, 0.7796704769134521, -1.281032681465149, -1.5765743255615234, -0.8384057283401489, 1.1261078119277954, -1.3544927835464478, 0.05403420701622963, 0.5326566696166992, 0.6253364682197571, -0.5079839825630188, 0.24487993121147156, -2.408005714416504, -0.10110951960086823, 1.55888831615448, -0.6421406865119934, 0.6401845812797546, -0.28210607171058655, -0.15161938965320587, 0.22821259498596191], [0.7518799304962158, -0.4689764678478241, 0.4093177616596222, -1.031588077545166, 1.5460715293884277, -0.4351431131362915, 0.23635220527648926, -0.8259977698326111, -0.7191938757896423, -2.0812668800354004, -1.1213651895523071, 0.02663501910865307, -0.4543096125125885, 1.6426688432693481, 0.5268880724906921, -0.8206309080123901, 0.32051584124565125, 1.3894643783569336, -0.014012088999152184, -0.06596928089857101, 0.7435781955718994, -0.2643091082572937, 0.7042590379714966, -1.1486098766326904, 0.9057807326316833, -1.4956331253051758, 0.12804050743579865, 1.5431551933288574, 0.07091818004846573, 0.22353874146938324, -0.8291012048721313, -0.3801538646221161, 0.5643236637115479, 1.6446363925933838, -0.026191383600234985, 1.6114614009857178, -1.281548023223877, -2.184676170349121, -1.4710817337036133, 0.35415831208229065, -0.984006941318512, -1.7600351572036743, 0.5984064340591431, -0.29542115330696106, -0.8846665620803833, -0.4108662009239197, -0.3823724389076233, 1.0310719013214111, -1.2993534803390503, 1.0792042016983032, 0.027167120948433876, 1.0299910306930542, 0.9028269052505493, -1.2782021760940552, 0.15147371590137482, -1.016635775566101, -1.2280491590499878, 0.39730021357536316, -0.3163796365261078, -0.2536153495311737, 0.8679581880569458, 2.492983102798462, -1.693945050239563, -0.41154465079307556, -0.3422507345676422, -0.05060511454939842, 0.6577377915382385, -0.0730310007929802, -1.915209174156189, 0.2475898414850235, 0.4487723708152771, 0.30548882484436035, -0.9701896905899048, -1.1060810089111328, -0.3079942464828491, -0.4801176190376282, -0.7910590767860413, -0.48510006070137024, -1.1105844974517822, 0.606666088104248, 1.5432909727096558, 0.28363868594169617, 0.2497396320104599, -1.0006929636001587, 1.4317954778671265, -1.6787282228469849, 0.2937394976615906, 0.5574087500572205, -1.3131210803985596, -0.23411637544631958, -0.4597451090812683, 0.4041261076927185, 0.8469780683517456, -0.19079576432704926, -0.2871488034725189, 1.9245452880859375, -0.7972925901412964, 1.365817666053772, -0.3414109945297241, 2.4433956146240234, -0.6003059148788452, -0.4148734509944916, 1.5986961126327515, -0.3071220815181732, 0.5083984732627869, 0.7437946200370789, 0.19036084413528442, 1.2450522184371948, 0.21456462144851685, 1.2197468280792236, -2.0098867416381836, -0.15034453570842743, 1.512665033340454, -1.8765017986297607, 1.5074572563171387, 0.5558115839958191, 0.7296565175056458, -1.4640085697174072, -0.23871593177318573, 0.29175546765327454, -1.3529345989227295, 0.26051732897758484, 0.4435264766216278, 0.34964561462402344, -0.9725136756896973, -0.6899081468582153, 1.576642632484436, 0.7510724663734436]]]], "dtype": "float32"}, {"__nd__": [24, 3, 0, 11], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[-0.8358948826789856, -0.7492488026618958, 0.12689907848834991, 1.1851311922073364, 0.6127730011940002, 1.0300109386444092, -0.2475164383649826, -1.4746100902557373, -0.3076491951942444, -1.3540141582489014, -0.01741357147693634, 0.7104607224464417, 0.743986964225769, 1.2159383296966553, -0.37574219703674316, 1.7730376720428467, -0.6298396587371826, -0.8099997043609619, 1.3114712238311768, -0.16689112782478333, 0.759976863861084, -0.17460942268371582, -1.0795549154281616, -0.36925777792930603, -0.5502085089683533, -1.2503336668014526, -0.07240678369998932, 0.2531836926937103, 0.8721968531608582, 0.031627412885427475, 0.2870963513851166, -0.11965277791023254, -1.407834768295288, 0.3393809199333191, -0.6044778823852539, -0.3752758204936981, 0.09365470707416534, -0.05480523034930229, 0.24419109523296356, 0.47811460494995117, -0.4022423028945923, -0.22132913768291473, -0.8609272241592407, -1.1454435586929321, 0.8583798408508301, -1.2696712017059326, -0.5515924096107483, -2.001511573791504, -0.3503507077693939, 0.5629974007606506, 0.828862726688385, -0.6430014967918396, 0.6845542192459106, 0.474409282207489, 1.6913611888885498, -0.21907733380794525, 0.08807986229658127, -1.074515461921692, -0.07362519204616547, 0.6619321703910828, 0.19219522178173065, 0.7998368740081787, 0.8641991019248962, 1.1674867868423462, -0.31378549337387085, 0.8354954719543457, -0.1051129475235939, 1.1606414318084717, -0.23459872603416443, 1.1590453386306763, 0.37426188588142395, 0.6766332983970642, -0.2140592634677887, 0.37270545959472656, 1.3318008184432983, 0.067112497985363, -2.1730778217315674, -0.550142765045166, 1.0972775220870972, -0.796725332736969, 0.192660391330719, 1.399979591369629, -0.5523452162742615, -0.13846564292907715, -0.2695726156234741, -0.3539893627166748, -0.862855851650238, 1.0139062404632568, 0.062389109283685684, -1.1985834836959839, 0.3213369846343994, -1.8788800239562988, 0.4877175986766815, -0.7066534161567688, -0.6254885792732239, -0.5174050331115723, -1.1453018188476562, -0.010731996968388557, -0.165344700217247, -0.6736624240875244, 0.8495471477508545, -0.886945903301239, 0.2338714450597763, -0.6815134286880493, -0.11415020376443863, -1.5705606937408447, 1.361753225326538, 0.12384113669395447, -0.41812261939048767, 0.016547173261642456, -0.8600168824195862, -0.8474650382995605, 0.1360420435667038, -0.0674365684390068, -1.1020166873931885, 0.9173263907432556, 0.02365153469145298, 0.021719204261898994, 0.29907476902008057, -0.33612555265426636, 0.2767537236213684, -1.1601570844650269, -1.2136235237121582, -1.0432636737823486, -0.3259442448616028, 0.8724785447120667, -1.8923425674438477, 0.8694883584976196], [0.475953072309494, 0.7543489933013916, -0.2656908631324768, 1.09140145778656, -0.21005778014659882, -0.001925238175317645, -0.8911086320877075, 0.9520604610443115, -1.5674688816070557, -1.1466108560562134, -0.9196233153343201, -0.4729580283164978, -0.9733282923698425, 0.23120331764221191, 0.029361408203840256, -1.3260077238082886, 1.064221978187561, -1.227325677871704, 1.3823225498199463, 0.426570326089859, 1.73186194896698, 0.11092525720596313, -0.05604387819766998, 0.2460622936487198, -0.9465541243553162, -0.6916224956512451, 0.12102403491735458, 0.0013792855897918344, 0.19772763550281525, -0.7457913160324097, 0.8810874223709106, -0.4906722903251648, -0.31989389657974243, 0.9924588799476624, 0.3268784284591675, 0.5251232385635376, 0.1655815839767456, 1.3261557817459106, -1.848161220550537, 0.4298744797706604, 0.2372571974992752, 0.4716958701610565, 1.2857685089111328, 0.37814027070999146, 0.7550340890884399, -0.22688992321491241, -0.7702657580375671, 0.3617601692676544, 0.6430854201316833, 0.6583463549613953, 1.231358528137207, 0.5129940509796143, -0.33848312497138977, -0.09275828301906586, -0.8830127120018005, 0.3729389011859894, -0.11771812289953232, -0.6894017457962036, -1.236525297164917, -0.9231254458427429, -0.13837873935699463, 0.07822265475988388, -0.9768457412719727, 0.08161275088787079, -0.17861466109752655, -1.553594708442688, -1.029341459274292, 1.207220196723938, 0.7264966368675232, -1.4010790586471558, 0.15713781118392944, -1.7605313062667847, 0.08741699904203415, -0.6859370470046997, 0.3659684956073761, 0.0728130117058754, 0.34829583764076233, 1.4574073553085327, 1.436722755432129, -1.8146132230758667, 0.5226561427116394, -0.8185413479804993, 0.012819930911064148, 0.7474116683006287, 0.08139919489622116, 0.15234485268592834, 1.4225889444351196, -0.9996383786201477, 0.6400807499885559, -0.042412951588630676, 0.9086902141571045, 3.0921316146850586, 0.7839266061782837, -0.6227477788925171, 0.17778541147708893, 0.32571348547935486, -0.9284422993659973, -0.39678850769996643, -1.227599024772644, -0.5220226645469666, 0.29403096437454224, -0.002542076399549842, 0.3906453847885132, 0.12325960397720337, -1.0832880735397339, 0.8353636264801025, -0.11951188743114471, 0.22909128665924072, 0.09944914281368256, -0.06456764042377472, 0.21610553562641144, 0.2895278334617615, -0.7339441180229187, 0.5580937266349792, -1.9439536333084106, -1.2451632022857666, -1.1867036819458008, -0.45725348591804504, 0.7424998879432678, -0.4061613380908966, 0.4357427954673767, 1.5378718376159668, -0.42252635955810547, 0.11576105654239655, 0.2395445555448532, 0.6546375155448914, 1.8070108890533447, 1.3621262311935425], [1.0170321464538574, 1.0588847398757935, 0.1576298624277115, 0.31457382440567017, -0.05325412377715111, -0.2535417377948761, -0.07476823776960373, 1.501128077507019, -0.4325109124183655, 2.0552031993865967, -0.91200190782547, 1.009992003440857, -0.3071470260620117, -2.296644449234009, -0.09220189601182938, -2.664930582046509, 0.5276841521263123, -0.7500361204147339, 1.8403046131134033, 0.8194742202758789, 1.1704113483428955, -0.915047824382782, -0.2876967787742615, -0.2610924243927002, 1.1965878009796143, -1.7552076578140259, 1.0831156969070435, -1.0702850818634033, 1.0192794799804688, -0.6227821111679077, 0.5544255375862122, -0.011243721470236778, -0.11010430753231049, 0.021101539954543114, -2.2308976650238037, 0.4533381164073944, -0.4465513527393341, 1.5376086235046387, -1.9656249284744263, 0.0851474180817604, -0.32152268290519714, 0.32729578018188477, 0.24739699065685272, 0.1621924489736557, 1.4150608777999878, -0.5473925471305847, 0.4818195104598999, 0.646834671497345, 0.24268026649951935, 0.338950514793396, 0.23008129000663757, -0.5670547485351562, -0.6546372771263123, -0.08235310018062592, 1.15676748752594, 0.025444742292165756, -0.33531203866004944, -1.551836371421814, -0.29619473218917847, -1.044950246810913, -0.9614076018333435, -1.1025323867797852, 0.07862978428602219, -1.1084822416305542, -1.5446137189865112, -0.5847607254981995, -0.6442530155181885, 0.6879547238349915, -0.08193289488554001, -0.5971438884735107, -1.4553658962249756, 0.5663564205169678, 2.6495141983032227, 0.20371940732002258, 0.05823130905628204, -0.15510576963424683, 1.5492053031921387, 0.803160548210144, 0.7869621515274048, -0.2982945740222931, -0.9721071720123291, -1.0223579406738281, -0.8558235168457031, -0.1408332735300064, -0.9858003258705139, 1.8677364587783813, 0.37501809000968933, 0.24249683320522308, 1.0055216550827026, 1.84867525100708, 1.022654414176941, 0.2883763015270233, -0.6203710436820984, -0.6212392449378967, -0.5177543759346008, 1.9103879928588867, 2.6343741416931152, 1.9302927255630493, 0.1788681596517563, 0.4414910674095154, 0.10566366463899612, 0.18255673348903656, 0.33930426836013794, 1.31462824344635, -0.03725894168019295, -0.5650988817214966, -0.2854514718055725, -0.6545971035957336, -1.3163471221923828, -0.630376935005188, -0.1667557805776596, -0.637470006942749, 0.36744335293769836, 1.3628727197647095, 0.7857458591461182, 1.8764839172363281, 1.9221380949020386, 1.071738600730896, 0.015195643529295921, -0.7534221410751343, 0.8261788487434387, -1.1575794219970703, 1.2828010320663452, 1.3683311939239502, -0.8397308588027954, -1.0934491157531738, -0.580411970615387, 0.6597511172294617], [2.317634344100952, 0.4641222059726715, 0.9332907199859619, -0.5360661149024963, -1.0702284574508667, 0.11500363796949387, 1.5246080160140991, -0.5622966289520264, 0.8424624800682068, -0.6624135971069336, -1.0593228340148926, 0.44524458050727844, -0.3431186079978943, -0.30487391352653503, -0.18611164391040802, 0.5328001379966736, 1.4353892803192139, -0.537550687789917, 0.2853083312511444, -0.35247907042503357, -0.8018783926963806, 0.63531494140625, -0.23842988908290863, 1.0919742584228516, -0.07244020700454712, -1.3542460203170776, 2.7090566158294678, 1.0176312923431396, -1.2833925485610962, 0.6442765593528748, 0.10303002595901489, 1.031751036643982, -0.512749969959259, -0.5006945729255676, -2.161614179611206, 0.6657407283782959, 0.6238330006599426, 0.41479358077049255, 0.41491928696632385, -0.7902692556381226, -0.4910655915737152, 0.5120121836662292, -0.9943680763244629, 0.15936970710754395, -0.11538366973400116, 0.13079099357128143, 0.5035451054573059, -0.13996610045433044, 1.443910837173462, -0.45123764872550964, -0.37701767683029175, -0.7570580244064331, 0.9585309624671936, 1.220083236694336, -1.5901026725769043, 1.1516510248184204, -1.4304732084274292, -0.47351136803627014, -0.24418345093727112, 0.791208803653717, -0.21931475400924683, -1.4088791608810425, 0.9834260940551758, -0.3443974554538727, -1.3010470867156982, -0.3052740693092346, 0.9544335603713989, -0.6038057208061218, 1.0449633598327637, -0.20446766912937164, 0.09590154886245728, -0.7396836876869202, 0.30066820979118347, -0.5951815247535706, 0.20383289456367493, -0.46082744002342224, -0.2779134213924408, 0.8948066234588623, 0.6435213685035706, 0.5052830576896667, 0.5961781144142151, -0.4084109663963318, 0.18045422434806824, -1.180620789527893, -1.4858564138412476, -0.11434130370616913, -1.1242398023605347, -0.174966961145401, -1.6320512294769287, 0.7659502029418945, 1.5470277070999146, -0.1309356987476349, -0.07261138409376144, 0.8913955688476562, -0.31845009326934814, 0.39228391647338867, -0.9550431966781616, 0.4827600121498108, -1.1792380809783936, -1.5468748807907104, 0.4015391767024994, 0.10242252796888351, 0.07902009040117264, -1.294763207435608, 1.574658751487732, 0.14065244793891907, -1.0498907566070557, -0.7985584735870361, 0.5035898089408875, -0.4505000412464142, -1.7926627397537231, 0.950519859790802, -0.3577258288860321, -0.4728473126888275, -0.6637775301933289, 0.673334002494812, 0.7693292498588562, -0.10257050395011902, 1.3591543436050415, -0.5894454121589661, 0.05225636810064316, -0.4220919609069824, 1.7713656425476074, 0.21103298664093018, 0.49404940009117126, 0.9447611570358276, -0.7879204154014587, 2.170342206954956], [-1.0148751735687256, 0.5727707743644714, 1.5644240379333496, 1.995041012763977, -0.41768041253089905, -0.734237790107727, -0.6324328780174255, -1.7894244194030762, -1.7096079587936401, 1.5937193632125854, -0.6309462189674377, 0.762289822101593, -1.9577338695526123, 0.02588329091668129, -0.6351637840270996, 0.7345538139343262, -0.4725660979747772, 0.4792718291282654, 2.103844404220581, 1.496184229850769, 0.7962386012077332, 0.7679896950721741, -0.20135480165481567, -1.2836992740631104, 0.882170557975769, -0.5300740003585815, -1.0026642084121704, 1.381676197052002, -0.18006379902362823, 0.4454599618911743, 0.7281136512756348, -1.3658428192138672, 1.6920766830444336, -0.30974432826042175, -0.7498294115066528, 1.0466532707214355, 1.463366985321045, -2.20920467376709, 0.07086391001939774, -0.5761843919754028, 1.876914143562317, 1.136135458946228, 0.7930099368095398, -1.729326605796814, -1.942081093788147, 1.1236237287521362, 1.317883014678955, 0.17524465918540955, -0.40370848774909973, 0.2752409279346466, -0.7176842093467712, -1.0016788244247437, 1.0890885591506958, 0.42833226919174194, 0.7157416939735413, 0.5090904831886292, 0.854495644569397, 0.24677976965904236, -0.21485556662082672, -2.340585947036743, 1.1770886182785034, -0.18818990886211395, -0.855545699596405, 0.972065806388855, 0.6308205723762512, 0.8921454548835754, -0.33571314811706543, 0.2963554263114929, -1.4297205209732056, -0.020335406064987183, 0.07883059978485107, 0.6987726092338562, -0.4364575445652008, 0.2342159003019333, -0.9097254276275635, -1.6085044145584106, -1.6873626708984375, 0.34892404079437256, -1.3627227544784546, -0.037638407200574875, 0.14211463928222656, 0.26392078399658203, -0.9471501111984253, -0.415115624666214, 0.22416965663433075, -0.4768959581851959, -0.07123823463916779, -1.8777039051055908, 2.5706005096435547, -0.02723683789372444, 0.9223883152008057, 1.7292853593826294, 3.0042386054992676, -0.8349770307540894, -0.7297531962394714, -0.3158067464828491, -0.810405433177948, -0.9677082300186157, 0.7128776907920837, 0.03276598080992699, 0.5983373522758484, 0.9802088737487793, -0.005059042014181614, 0.30067679286003113, 0.4336963891983032, 0.06146116182208061, 0.6816322803497314, 0.18933601677417755, -1.594251275062561, 1.8516802787780762, -0.7723247408866882, 0.2760610282421112, 0.4683857262134552, -0.520218014717102, -0.12261997163295746, -0.6702357530593872, 0.10955989360809326, 0.12409321218729019, -0.8868231177330017, 0.5251514315605164, 0.059859029948711395, 0.5234819054603577, 1.071797490119934, 0.431262344121933, -0.43348610401153564, 0.13297565281391144, 0.2226899117231369, 1.4318784475326538], [-1.6578388214111328, 0.29400110244750977, 0.0339222215116024, -0.8935646414756775, 0.893549919128418, 0.31426891684532166, -1.5266846418380737, 0.097648985683918, 2.5166265964508057, -2.1129019260406494, 0.39228716492652893, -0.2816097140312195, 1.5257277488708496, -0.47934406995773315, 0.032824426889419556, 0.14955087006092072, 0.7729616165161133, -0.38365814089775085, -0.08329722285270691, 1.364730715751648, -1.0578181743621826, -0.6077969670295715, -0.6904796957969666, -0.37923958897590637, 0.701796293258667, 0.7651252150535583, 0.9863858819007874, 0.7311770915985107, -1.2199931144714355, -0.5524089336395264, -0.6064444184303284, -0.7909141182899475, -0.12936373054981232, 1.8565397262573242, 0.39214202761650085, -0.3200754225254059, 0.9244135618209839, -1.0805565118789673, -0.4779898226261139, -0.616401731967926, -0.5337314009666443, -0.20328645408153534, 1.3433773517608643, -0.3232789635658264, -0.13890333473682404, -1.4337339401245117, -1.0357264280319214, 1.0068706274032593, 0.42320600152015686, -0.7944551110267639, 0.02751559391617775, 0.7661355137825012, -1.585702657699585, 0.23805958032608032, 1.0970360040664673, 1.532659888267517, 1.1489118337631226, -0.6120585799217224, -0.6388119459152222, 0.11400896310806274, -1.455405592918396, -0.8001040816307068, 0.4514569640159607, -1.2543673515319824, 1.549737572669983, 0.5399158596992493, 1.3486632108688354, -0.33256298303604126, -0.9248870611190796, 0.2773328423500061, -1.8104209899902344, -0.2766287922859192, 0.8609151840209961, -0.20586316287517548, -0.519418478012085, 0.21517786383628845, 0.9052873849868774, -1.4424010515213013, 2.438474178314209, 0.545536994934082, 0.0576489083468914, 0.21373111009597778, 0.9957472085952759, 0.7587278485298157, -0.6985112428665161, -0.9697425961494446, -0.4575822949409485, 0.05077027529478073, 0.7379735112190247, 0.13245674967765808, 1.2510640621185303, 0.5004023313522339, -0.2330016940832138, -0.07991482317447662, -0.6538897156715393, -0.8810664415359497, -0.6469339728355408, -1.1413915157318115, -0.6272270679473877, -0.34584030508995056, 1.1056169271469116, -0.026563094928860664, 1.852360725402832, 2.0832159519195557, 0.15319998562335968, -1.4447638988494873, 0.4346430003643036, 0.12737280130386353, -0.16484132409095764, 0.22960589826107025, -0.6053653955459595, 0.018273279070854187, 0.6618918776512146, 1.3784006834030151, -0.2590155303478241, -0.11556745320558548, 1.3950185775756836, -0.6639583706855774, -1.0620027780532837, 0.256256639957428, -0.5868607759475708, -0.610596776008606, -1.8399771451950073, -0.19949372112751007, 0.9648222327232361, 1.4648369550704956, 0.45386284589767456, -0.2558766305446625], [0.377704381942749, -0.2380003184080124, 0.2873415946960449, -1.1908379793167114, 0.9592229723930359, -0.046990517526865005, -1.028624176979065, 0.45729508996009827, 1.3457969427108765, 0.32127705216407776, 0.4814501702785492, 0.2634153664112091, 0.6671515703201294, 0.6821397542953491, 0.19687587022781372, 0.8088603615760803, 0.04119270667433739, -0.6221162676811218, -1.339422345161438, 0.7266095280647278, -0.7369012236595154, -0.833522617816925, 0.3589756488800049, 1.4004418849945068, 1.0090843439102173, 0.48459887504577637, 1.268250584602356, -1.908002257347107, -0.526794970035553, 1.2722041606903076, -0.17821629345417023, 0.8138281106948853, -0.25294092297554016, -1.0799450874328613, 0.4599095582962036, -0.681571900844574, -0.5159640312194824, 0.8902517557144165, -0.20690129697322845, -1.4892030954360962, 0.727898895740509, 1.3066072463989258, -1.629830241203308, 0.7036844491958618, -0.7722615599632263, -0.8739343881607056, 2.194603443145752, 0.4359895586967468, 0.739591121673584, -0.49979451298713684, -1.1964937448501587, 0.13110433518886566, -0.5122163891792297, -0.31255486607551575, 0.7013121843338013, -0.352039098739624, -0.7527945637702942, 2.1542398929595947, -0.37460216879844666, 0.6330841183662415, -0.46503421664237976, 1.7002875804901123, -0.9740995168685913, 0.628663182258606, -0.06693953275680542, 0.07772330939769745, -0.9790493249893188, 0.2913842797279358, -0.42835837602615356, 0.7645682096481323, -0.6208759546279907, -1.0353978872299194, -0.18148095905780792, 1.1180344820022583, -0.049602195620536804, -0.7008435726165771, -0.6257852911949158, -0.6792227029800415, 1.728961706161499, 0.7088293433189392, 0.3301137685775757, -0.9619460701942444, -0.5938128232955933, 0.7794328331947327, -1.2300145626068115, 0.6742097735404968, -0.383110374212265, 0.19750681519508362, 0.6440991163253784, -0.22438964247703552, -0.8863945007324219, 1.9284522533416748, 1.6695408821105957, -2.99130916595459, -0.2283172756433487, -0.5240658521652222, 1.1552497148513794, -1.1833778619766235, 0.5871095061302185, 0.7604048848152161, 0.453437864780426, 0.27427998185157776, -0.25279560685157776, 1.8582361936569214, 0.3568568527698517, 0.5719924569129944, -0.9958747029304504, 0.5415090322494507, 0.9722817540168762, -0.13116605579853058, -0.8673829436302185, 0.7029394507408142, -0.15922020375728607, -0.38762351870536804, -1.1842714548110962, 0.282979816198349, -0.5516008734703064, 1.320663332939148, 0.19337205588817596, -0.5382242798805237, -0.05696496739983559, -0.5780189633369446, -0.3622652590274811, 0.7867183089256287, 0.007981694303452969, -0.050870634615421295, 0.5986065864562988, 1.3759996891021729], [-0.03144845739006996, -0.3479301929473877, -1.0073498487472534, 0.2890443801879883, 0.3540019392967224, -0.08893460035324097, -1.0436091423034668, -0.5608684420585632, 1.1219911575317383, -1.6510590314865112, 1.1139763593673706, 0.5492724180221558, -0.7973261475563049, 0.6113355159759521, 2.1136956214904785, 1.3068090677261353, 0.6670776605606079, -1.0916715860366821, 1.8940644264221191, 0.15504568815231323, -2.6306841373443604, 0.6888604760169983, 0.24163898825645447, -0.06794705986976624, -0.38706716895103455, -0.001329214544966817, -1.1946921348571777, -0.03842289745807648, -0.7059695720672607, 0.983269453048706, -1.9312294721603394, 1.1229820251464844, 1.3213541507720947, 1.8286609649658203, 0.546682596206665, 2.1593523025512695, 1.5258740186691284, -0.4537758529186249, 0.3329174220561981, -1.4556770324707031, -1.7724369764328003, 0.4028996229171753, 0.3836289048194885, -0.8109895586967468, -1.153381109237671, -0.869113564491272, 0.39523056149482727, 0.35115525126457214, 0.12550264596939087, -0.7370885014533997, 0.9446954727172852, 1.354262113571167, -1.2691246271133423, 0.41972988843917847, 0.2748650014400482, -0.41738012433052063, 0.062212925404310226, -0.5591364502906799, -0.5670019388198853, -0.17055334150791168, -0.5414919853210449, -0.6436651945114136, -0.9240917563438416, -0.3895909786224365, 0.027925703674554825, -1.2624238729476929, 0.6948047876358032, -1.4214355945587158, 1.6696504354476929, -1.105216383934021, -2.6395578384399414, -0.8920459747314453, -0.8474315404891968, -0.05741894990205765, 0.38918450474739075, 1.290188193321228, -0.28656068444252014, -0.6804601550102234, 1.910409927368164, 0.7121158242225647, -0.19764645397663116, 0.21351002156734467, -0.5051677823066711, 0.7244899272918701, 0.23529528081417084, -0.9729953408241272, -1.2811263799667358, 0.7115157246589661, -0.04221860319375992, -0.4167034924030304, -1.0354317426681519, -0.28475499153137207, 0.4809953570365906, -1.703911542892456, 2.4613072872161865, -0.8967798352241516, 1.8159680366516113, 0.10917644947767258, 1.5352399349212646, -0.2865552604198456, -1.158981204032898, -0.2943342626094818, -0.41131168603897095, 0.09129966050386429, 1.0535980463027954, -1.0243494510650635, 0.5897109508514404, 0.9499473571777344, 0.19140397012233734, -0.6202195286750793, 1.286179542541504, 1.198706030845642, 0.9827528595924377, 0.25231993198394775, 1.6330840587615967, -1.313454031944275, 0.6936138868331909, 1.5228016376495361, 1.2578656673431396, -1.431409478187561, 0.7361136674880981, 1.8885295391082764, -0.4843498170375824, -1.698805332183838, -0.9683853387832642, 0.5747280120849609, 1.3193913698196411, -1.2428131103515625], [0.95982426404953, -0.6710734367370605, -1.226505160331726, -0.3398974537849426, -0.21387100219726562, -0.3578338921070099, 1.3507121801376343, 2.0803439617156982, -0.13370034098625183, -0.2339482605457306, -1.749214768409729, 0.7462241649627686, -0.5563958883285522, -0.3117798864841461, 0.8527589440345764, -1.3016209602355957, -0.1770579069852829, -1.9441509246826172, 0.28382429480552673, 0.7264275550842285, -0.27921757102012634, -0.27317413687705994, -1.0340163707733154, -1.0922547578811646, 2.2001311779022217, -1.467574119567871, -0.42971083521842957, -1.0709656476974487, -1.9145039319992065, 0.5608466267585754, -1.138522744178772, 1.2942264080047607, 0.6678308844566345, 1.492601752281189, 0.10418812185525894, -0.08671346306800842, -0.6209465861320496, 0.5910029411315918, -0.6406233906745911, 1.6605756282806396, -0.9725055694580078, 0.6361387372016907, -0.03333841264247894, -0.26799967885017395, -0.32760220766067505, 1.038413643836975, 0.8235840201377869, -0.02223372459411621, -0.06741315126419067, -0.7484764456748962, -0.7365787625312805, -0.09406916052103043, -0.5889838337898254, -1.5563621520996094, -0.3202938437461853, 1.6770884990692139, 0.751896321773529, -0.06607294082641602, 1.0512661933898926, -0.5021456480026245, 0.3442901074886322, -0.49515217542648315, -1.5327099561691284, -0.1248701885342598, 0.3986944556236267, 1.3062658309936523, -0.404079407453537, 0.6713934540748596, -0.49285387992858887, 0.8091824650764465, -0.4801160991191864, 1.1916245222091675, -0.5335688591003418, 0.5650676488876343, -0.49573567509651184, -0.8254448175430298, -0.9492902755737305, -1.0713891983032227, 1.6466501951217651, -0.5980514883995056, 0.2677907347679138, 1.1912490129470825, -1.2765206098556519, 0.09571308642625809, 0.05789751932024956, -0.08764170110225677, 0.1090737134218216, 0.5822839736938477, 1.1477991342544556, -0.4403032660484314, -0.08330415189266205, 0.6194721460342407, -0.32200273871421814, -0.5299765467643738, 0.9632738828659058, 1.219870924949646, 0.20568180084228516, 0.30141404271125793, -0.06572018563747406, 0.7685028910636902, -0.6470226049423218, -0.3129523694515228, 0.7508363127708435, 0.025832049548625946, -1.4275296926498413, 0.4752689003944397, 1.1094118356704712, 0.23114220798015594, 0.4294479191303253, 0.7382975816726685, -0.5149836540222168, 0.8096388578414917, -0.3510623872280121, -1.0460355281829834, -0.7116126418113708, -0.11534764617681503, 0.36828693747520447, 0.8948222398757935, -1.630694031715393, -0.7024412155151367, -0.8779889941215515, 1.8513773679733276, -0.2612711787223816, 0.2793801426887512, 0.7146572470664978, 0.3177106976509094, 1.13005793094635, -1.1528215408325195], [0.7703520059585571, 1.3324954509735107, 0.9360408782958984, 0.41417816281318665, -1.7768007516860962, 1.5103065967559814, 1.5747824907302856, 0.3676993250846863, -1.885646939277649, -0.05939231812953949, -1.344782829284668, -0.032565124332904816, -0.17391858994960785, 1.2989064455032349, -1.2855966091156006, 0.14010469615459442, -0.8834280967712402, -0.7489727139472961, 0.28396138548851013, 2.0690367221832275, -0.30225709080696106, -2.028686046600342, 1.3782014846801758, -0.13622251152992249, -1.1665276288986206, -0.6686456203460693, -1.0978144407272339, 1.2967798709869385, 0.20340681076049805, 0.39973023533821106, 1.3061437606811523, 0.9924890995025635, 1.3622742891311646, -0.7001648545265198, -0.5736575722694397, -0.8392177224159241, -0.30907899141311646, 0.05053792521357536, -2.5709218978881836, 0.8998321890830994, 0.06359683722257614, 0.5740432739257812, -0.19338852167129517, 0.9018384218215942, 0.6780436038970947, 1.2971750497817993, -1.8571518659591675, -0.3134533166885376, 0.10926979035139084, -0.7485412359237671, -0.041685860604047775, 0.4360748827457428, 0.9507409930229187, 0.08263125270605087, -1.5193216800689697, 0.024291792884469032, -0.7571995854377747, -0.1973535716533661, 0.40623563528060913, 0.4869012236595154, 0.3206127882003784, 0.9890263080596924, -0.0042726825922727585, 0.9661753177642822, -1.625704050064087, 2.329393148422241, 1.5019418001174927, 0.9289839267730713, -1.9290554523468018, 0.8765773177146912, 0.030877448618412018, 0.7006205916404724, -1.1638448238372803, -1.0311291217803955, 0.6532650589942932, -0.5051576495170593, -1.165086030960083, 0.29063379764556885, -0.46513256430625916, -1.4899908304214478, 0.22223906219005585, -0.1401837021112442, 0.8350504040718079, -0.3750828504562378, 0.2734629809856415, -0.17630359530448914, -0.472491592168808, -0.04398516193032265, 0.674040675163269, 0.2818588614463806, -0.5247704982757568, -0.32427600026130676, -0.31513869762420654, -0.38651368021965027, 0.49218615889549255, -0.3475703299045563, -0.7179967164993286, 0.6995590925216675, 1.1179192066192627, -0.9174093008041382, 0.27545106410980225, -0.6332220435142517, 0.8225661516189575, -1.6808053255081177, -1.666079044342041, 1.0115206241607666, 0.9779115915298462, 0.980930745601654, -0.875335693359375, -0.19932761788368225, 0.407335489988327, -1.306712031364441, -0.5254254937171936, -1.0679564476013184, -0.7849779725074768, 0.7246192097663879, -2.331592559814453, -1.2546402215957642, 0.5330635905265808, 0.12917938828468323, -0.41358068585395813, -0.24865184724330902, 0.5803433656692505, 0.708893895149231, -0.41416412591934204, 0.300778865814209, -1.5648088455200195, -0.29681456089019775], [-1.7853127717971802, -0.15081332623958588, -0.4000321626663208, 0.049646053463220596, -0.6624380946159363, -0.9702860713005066, -1.1625686883926392, 0.23886623978614807, -0.7224103808403015, 0.31410738825798035, 0.9311237335205078, 1.6793855428695679, 2.167529582977295, 0.1512748748064041, 0.31447890400886536, -0.270940363407135, 0.4461929500102997, 0.9314620494842529, 0.3223262131214142, 0.3966730833053589, -0.7367970943450928, -0.2794441878795624, 1.3225395679473877, -2.5148329734802246, -0.07270157337188721, -0.5907094478607178, 0.5183554887771606, -0.4526333808898926, 0.05946575105190277, -0.7407355904579163, 0.025952471420168877, -0.08140141516923904, -0.47028255462646484, -0.36385494470596313, -1.6922768354415894, -0.26053372025489807, -1.017321228981018, -0.38105809688568115, -1.6402114629745483, 0.19181686639785767, -0.739232063293457, -0.17645694315433502, -1.355384111404419, -0.2514919936656952, 0.8333863615989685, -0.3275006413459778, -0.8390824794769287, -1.1766612529754639, 1.6503393650054932, 0.8562300801277161, 1.4443542957305908, 0.4559018015861511, 0.7283895015716553, -0.6010465621948242, 1.1884969472885132, -0.09783896803855896, 0.3237132728099823, -0.3091692328453064, 0.6899526715278625, 0.49248161911964417, -0.7248042821884155, -0.04111713543534279, 0.15855205059051514, 0.44541317224502563, -0.20097893476486206, 1.4427436590194702, 0.9577332139015198, -1.2005105018615723, 0.4760059118270874, -0.29688760638237, -0.8435278534889221, 0.9033945798873901, 0.7320290207862854, 1.3386262655258179, 0.32177233695983887, 1.2957810163497925, -0.02166183479130268, 0.9606038928031921, 0.5161052346229553, 0.7067704796791077, 0.6589914560317993, -0.3497709333896637, -0.9404901266098022, -1.027974009513855, -0.592355489730835, -1.8310673236846924, -0.5050987601280212, 1.6315999031066895, -0.4798761308193207, 1.8481477499008179, 0.3542715907096863, 0.667157769203186, 0.5273805260658264, 2.598446846008301, -1.2199510335922241, -0.1264101266860962, -0.06621861457824707, 0.9506892561912537, 2.2723617553710938, 0.8597820401191711, 0.6084009408950806, -0.3769833445549011, -0.6140115857124329, -0.6302647590637207, 0.4164595901966095, 0.9110977649688721, -0.4488503038883209, -0.6631534099578857, -0.15684349834918976, -0.9585694670677185, -0.7046802639961243, -1.1722886562347412, -1.9107766151428223, 1.3526983261108398, 1.2447832822799683, -0.17372524738311768, -1.1514196395874023, -0.641072690486908, -0.8582776188850403, 0.6328040361404419, -0.3100089430809021, -0.12106157094240189, 0.1411079615354538, 0.059718601405620575, -0.3042883276939392, -0.4676593542098999, 1.924280047416687, 0.5882667303085327], [0.22201497852802277, -0.825310230255127, 1.0493254661560059, 0.4837816059589386, 0.2771356999874115, 1.1302882432937622, 0.4466073215007782, 1.3730850219726562, 1.0600801706314087, -0.35941261053085327, 0.29871442914009094, 0.11735375970602036, -0.46270042657852173, 0.21547754108905792, 1.881414532661438, 0.5977687239646912, 0.22941482067108154, 1.2952392101287842, -1.3581748008728027, 0.05223029479384422, 1.2087210416793823, 0.786665678024292, -1.2966890335083008, 1.0583134889602661, -0.5034135580062866, -0.417125403881073, -1.8627322912216187, 1.4723920822143555, -0.05572802573442459, 1.3327666521072388, 1.1291345357894897, 1.402080774307251, 0.07086050510406494, 0.4109205901622772, 1.7463186979293823, -0.05157484486699104, 0.05795607715845108, 0.06178279221057892, 0.922940194606781, 0.5914769172668457, 2.0046579837799072, -0.4803936183452606, -0.029344480484724045, -1.089215636253357, -0.21820957958698273, 1.2439682483673096, -1.0281904935836792, 0.8043040633201599, -1.0205543041229248, -0.1639208197593689, -2.1437246799468994, -0.037690624594688416, -0.5866401195526123, 0.9661582708358765, 2.6291513442993164, -0.8946570158004761, 1.1091402769088745, 0.2503519654273987, -1.0028581619262695, -1.1750417947769165, -0.35496246814727783, 1.8394386768341064, -1.494105339050293, 1.3088834285736084, 0.7413603663444519, -1.2675155401229858, -0.8938004970550537, 0.027823535725474358, -1.6748294830322266, 0.8567380309104919, -1.1375222206115723, 0.9724658727645874, -1.9346505403518677, 0.9344436526298523, -0.502798318862915, 0.07816492021083832, 0.8372974395751953, -1.3022775650024414, -0.32449743151664734, -0.2747761905193329, 1.9095745086669922, 1.369592547416687, 0.06727497279644012, 0.31386882066726685, -0.6827303171157837, 3.1868093013763428, 0.48806527256965637, 0.23080043494701385, -2.104165554046631, -1.099239468574524, 0.8684420585632324, -0.3566846549510956, 0.41742822527885437, -0.6272463202476501, -0.023271147161722183, -1.925707459449768, -1.831377387046814, -0.8740531206130981, -0.009234629571437836, -1.2198766469955444, -3.2064292430877686, -0.007831145077943802, -0.7207728624343872, -1.4791213274002075, -1.7304842472076416, 0.49644073843955994, 1.4909183979034424, 0.2844180166721344, 1.5888993740081787, -1.5118601322174072, 0.30797621607780457, -1.5110819339752197, 0.4698239266872406, -0.5166385173797607, -1.7970994710922241, 1.041196584701538, -0.7534015774726868, 0.25100523233413696, -0.2050403654575348, 0.567503035068512, 0.05643298476934433, 0.6638681888580322, 1.3000216484069824, -1.7656530141830444, 0.2509862780570984, 0.9761643409729004, -0.056988675147295, -1.5491487979888916], [-0.46830376982688904, -0.26096558570861816, -0.10488025099039078, -1.2206637859344482, -0.009622384794056416, 0.49300307035446167, 0.05860695242881775, -1.0403059720993042, 1.8297832012176514, 1.3271164894104004, -2.016213893890381, -0.6377583742141724, 1.3280575275421143, -1.1335264444351196, 0.5406962037086487, 0.9593963027000427, -2.0555036067962646, 0.8223453760147095, 0.9657633900642395, -0.8715633153915405, -0.1449470967054367, -0.7353242635726929, -0.28206557035446167, -0.9480443596839905, 1.4154942035675049, -0.6959812045097351, -0.9025841355323792, -0.2192663550376892, -1.6755030155181885, -0.23684659600257874, -0.18454201519489288, -1.0751020908355713, 0.3577481806278229, -0.5504014492034912, -0.9288245439529419, 3.518751382827759, 0.4594692289829254, 0.22687789797782898, -0.18868482112884521, 1.1220850944519043, -1.5147383213043213, 1.6346534490585327, 0.15004830062389374, 0.7452393174171448, -0.6045106053352356, -0.7560360431671143, 1.197738766670227, 0.6030040383338928, -0.08708606660366058, 0.6175930500030518, -1.0120733976364136, 2.7930169105529785, 0.12339857220649719, 0.45719876885414124, -0.06835391372442245, -0.13776984810829163, 0.7231460213661194, 0.1330215036869049, 1.1748722791671753, -0.531264066696167, -0.6107739806175232, -0.6991296410560608, -0.13000397384166718, 0.7378265261650085, 1.3347147703170776, -0.49625423550605774, -0.16696158051490784, 1.1136778593063354, 1.6219419240951538, -0.43126246333122253, 0.040963511914014816, 0.9920684695243835, -1.9328572750091553, -0.41132625937461853, 0.7708724737167358, 0.015872841700911522, -1.0139471292495728, -0.9995001554489136, 1.1484726667404175, 0.8885493278503418, -0.09056151658296585, 1.2929697036743164, 0.27659356594085693, 1.1859238147735596, 0.964833676815033, 0.587320864200592, -1.2859059572219849, 0.030101845040917397, 1.2193812131881714, 0.12335680425167084, 0.0035535763017833233, 0.06500809639692307, -1.3914680480957031, -0.25070858001708984, 1.0160670280456543, -0.5895722508430481, -1.3754440546035767, -0.9886310696601868, 0.24861331284046173, 1.433512806892395, 1.3255211114883423, -1.5125471353530884, 1.3714091777801514, 1.7262159585952759, 0.7102624773979187, 0.5017229318618774, -1.120277762413025, -1.8966102600097656, -2.1004607677459717, 1.2690699100494385, -0.419701486825943, 2.2115068435668945, 1.2436772584915161, 0.4920459985733032, 1.715101718902588, 0.3748534023761749, -0.9492154717445374, -0.0963742807507515, -0.10158959031105042, -0.21408817172050476, 0.4978778660297394, -0.30592212080955505, 2.254148483276367, -0.02783900313079357, 0.5233485698699951, 0.33845528960227966, 2.0880579948425293, -0.8802734613418579], [-0.09685534238815308, -0.42455676198005676, -0.06631786376237869, 0.4653554856777191, 0.07762712240219116, 0.875800371170044, 0.9099562764167786, 1.8397501707077026, 0.13544268906116486, -1.1709829568862915, 0.6720257997512817, -1.1553623676300049, 1.2752114534378052, 0.5028891563415527, -0.5226072072982788, 0.26588839292526245, 0.5414416790008545, -1.908053994178772, 0.7178407907485962, 0.8176060914993286, 1.1672877073287964, -1.0831577777862549, 0.1908620297908783, 0.6445098519325256, -0.09629300981760025, 0.10035308450460434, -2.7999343872070312, -0.8576048612594604, 2.095841884613037, -0.37961193919181824, 0.5303339958190918, 0.7724264860153198, -1.789976716041565, -0.43304023146629333, 0.019092099741101265, -0.8704396486282349, -0.405965119600296, -0.8830041289329529, -0.03694120794534683, 0.20337386429309845, -0.15640291571617126, -1.0084367990493774, 0.7341527938842773, -0.5101417899131775, -0.9838986396789551, 0.5265333652496338, -1.2068284749984741, -0.6122155785560608, -0.19389398396015167, -0.092678964138031, 0.6380778551101685, 0.025107506662607193, -0.4135475158691406, -1.0107030868530273, 0.6980884075164795, -1.7849371433258057, -0.7097217440605164, 0.0020787238609045744, 0.6126374006271362, 0.07265441119670868, -1.631847858428955, -0.2776798605918884, -0.21648600697517395, -0.5280066728591919, 0.4089539051055908, 0.5135959982872009, -1.1121610403060913, -1.6571325063705444, -0.7844823598861694, -0.2070075422525406, -0.22883255779743195, -0.7988958954811096, -0.13902176916599274, 0.35767295956611633, 1.1424999237060547, -1.2609375715255737, 0.16098633408546448, 0.001285675447434187, 0.17974241077899933, 0.5583406686782837, 0.4944646954536438, 1.9739692211151123, -1.740859031677246, 0.16776426136493683, -1.9414232969284058, -0.2284032553434372, 0.3443758487701416, 0.5194586515426636, 0.3407600224018097, 0.7003224492073059, -0.7299378514289856, -0.6058750152587891, 1.398093342781067, -1.0266343355178833, 1.1197314262390137, -0.4151467978954315, 0.401166170835495, -0.6226645708084106, 0.1861235797405243, 0.5935116410255432, -0.5146067142486572, -1.1685324907302856, 0.01690761372447014, 0.8996232151985168, -0.21726611256599426, -0.09879586100578308, -1.6794990301132202, 0.35269972681999207, 0.5984217524528503, -0.7314910292625427, 0.8741938471794128, 2.183398485183716, -1.2591501474380493, -0.8338804841041565, 1.003108263015747, -0.3480583429336548, 2.023637294769287, -0.14441116154193878, 1.0303740501403809, -0.8925147652626038, -0.4724208116531372, 1.9606989622116089, -0.25419896841049194, 0.8153796195983887, -0.20321476459503174, -0.705878734588623, -1.8100229501724243, -0.5055094361305237], [0.714831531047821, -0.3173018991947174, -1.2941776514053345, -1.4639673233032227, -1.0854027271270752, 0.6021865010261536, -0.2567180395126343, -0.4536289572715759, -0.31264811754226685, -0.4917534291744232, -0.5720130205154419, 1.3101252317428589, -1.8691959381103516, 0.28621914982795715, 1.0683305263519287, -0.24423938989639282, -0.29459908604621887, 0.20619729161262512, -0.2657933235168457, -1.329913854598999, -0.4762304127216339, 1.0662142038345337, -1.0366237163543701, 1.6454713344573975, 0.8234358429908752, -0.46964120864868164, -0.8931422829627991, 1.7702890634536743, -0.21018445491790771, -0.7250217199325562, 0.3537501096725464, 0.3645409643650055, -0.305979460477829, -0.5063034296035767, -0.07839080691337585, 1.1358263492584229, -0.41713929176330566, -1.003495693206787, 0.45921552181243896, -0.32046425342559814, 0.5244039297103882, -1.4868748188018799, 0.11626992374658585, 1.3827509880065918, -0.3529502749443054, 0.3415294885635376, 0.05350082367658615, 1.0003392696380615, -0.029584065079689026, -0.3459717631340027, -0.7965437173843384, -1.7697259187698364, -1.3170711994171143, 1.0991003513336182, -0.591560959815979, 0.9867937564849854, -0.3804602324962616, 1.3834749460220337, 1.6948285102844238, 0.575731098651886, 0.859587550163269, -1.4052660465240479, -0.6975130438804626, 1.2662943601608276, 1.767187476158142, 1.728770136833191, 0.2546926736831665, 0.15920689702033997, 0.5027711987495422, 0.09844600409269333, 0.05021010339260101, 3.049191474914551, -1.2106220722198486, 0.19160282611846924, 1.6242462396621704, 0.13392896950244904, -0.31876906752586365, -0.665732741355896, -0.40941521525382996, 1.2953684329986572, -1.8049763441085815, 2.765833854675293, -0.5225287079811096, -1.577609896659851, 1.2620524168014526, 0.49012601375579834, 2.7688348293304443, -1.3849384784698486, 0.025090020149946213, -1.052754521369934, -1.8083442449569702, 0.25578877329826355, 0.11071525514125824, -1.7096682786941528, 0.4000265598297119, -1.037412405014038, -1.3169554471969604, -1.2384299039840698, -0.9096704125404358, -0.8472113013267517, 0.5423204898834229, 0.22807036340236664, -0.3610687255859375, -0.8458987474441528, 0.47804179787635803, 0.16292904317378998, 0.8911266326904297, 0.6079750657081604, -2.151214122772217, 1.2050827741622925, -0.41306307911872864, -1.4391167163848877, -1.4050074815750122, -2.9402096271514893, 0.24221239984035492, -0.4347976744174957, 0.3499106168746948, 0.04636004567146301, 0.6569357514381409, 2.0511252880096436, 2.5112674236297607, 1.8448987007141113, 0.44662418961524963, 1.9522531032562256, -0.617970883846283, 1.3505008220672607, -0.9971863031387329, 0.2233327478170395], [-0.5478591918945312, -1.9262688159942627, -0.8238412141799927, 0.13340675830841064, 0.1508740782737732, -0.22408032417297363, -0.044701989740133286, 0.26849234104156494, 0.8815132975578308, -0.14826415479183197, 1.27024507522583, -0.08308230340480804, -0.5193064212799072, 0.6252172589302063, -0.4810872972011566, 0.16924957931041718, 0.15029126405715942, 0.040379561483860016, 0.6857789158821106, -0.21452437341213226, 1.031296730041504, -1.3047051429748535, -0.07950279861688614, 2.663667917251587, -0.700077474117279, 0.8567817807197571, 0.3787931203842163, 0.030606135725975037, -0.6618329882621765, -1.7467548847198486, -0.19473101198673248, 1.284257173538208, 0.5719641447067261, -0.4866505265235901, -2.1547534465789795, -0.8815433979034424, -1.5028785467147827, -0.671709418296814, 0.5039470195770264, 1.0365077257156372, -0.14219510555267334, -0.4622913599014282, 1.236851453781128, 0.4361576735973358, 0.384895384311676, -0.5007978677749634, -0.17428675293922424, 1.5182944536209106, -0.5330233573913574, -0.26464107632637024, -2.278564214706421, 0.78206467628479, 0.5159997344017029, 1.7313168048858643, 0.05596008151769638, -0.13113009929656982, 1.2935315370559692, 0.09583813697099686, 0.3917151987552643, -0.7229489088058472, -0.17037959396839142, 0.18915262818336487, 2.319948196411133, 0.5499929785728455, -1.0879515409469604, 0.22610539197921753, -0.6433438658714294, 0.8096803426742554, 0.48140373826026917, -0.03988303616642952, 1.100003957748413, 0.8616591095924377, 1.2388311624526978, -0.5470274686813354, 0.37980592250823975, 1.3019782304763794, -1.422545313835144, -0.8051163554191589, 1.6887257099151611, 0.2684843838214874, -0.040050532668828964, 1.0474363565444946, 0.5620158314704895, -0.4296327233314514, 0.6709493398666382, -1.931365966796875, -1.4202057123184204, 1.0117751359939575, -0.5158944725990295, 0.6588932275772095, -1.062092900276184, -0.4752914309501648, 0.10896942764520645, -0.1622089296579361, -0.4149434566497803, 0.013868732377886772, 1.8453311920166016, 0.5282554626464844, -0.7254921793937683, 0.7150225043296814, -1.1059908866882324, 1.449720859527588, -1.1551854610443115, -1.3830605745315552, -0.7547379732131958, -0.035617150366306305, -0.07616342604160309, 1.1730700731277466, 0.15202774107456207, -0.05107024312019348, -0.8404272794723511, -0.009613746777176857, -0.79011470079422, -1.4587125778198242, 1.376145839691162, 1.1739821434020996, -0.013660840690135956, 1.2414034605026245, 0.7518717646598816, 0.2850859463214874, 1.2567987442016602, -0.8305980563163757, 0.33954688906669617, -0.6330119371414185, 0.007817374542355537, -1.130264163017273, -0.8382492065429688, 0.08776146918535233], [-0.6402801275253296, -1.4363415241241455, -1.7293576002120972, -0.540558934211731, -0.9613327383995056, -0.6453918218612671, 0.16350992023944855, 0.9301466345787048, 0.33634206652641296, -0.34599733352661133, 0.0665067508816719, -1.1745121479034424, -0.5947383046150208, 0.11392145603895187, -0.19709721207618713, -0.4981481432914734, 0.8895739912986755, 0.40673720836639404, 0.8836843967437744, -0.8154907822608948, 0.4991302788257599, 0.6931819319725037, 1.4562242031097412, -0.3765331208705902, 0.5969178080558777, 0.98912513256073, 0.00554424524307251, -0.27162349224090576, 1.3366196155548096, 0.32621607184410095, 0.21104952692985535, 0.15316589176654816, -0.46222618222236633, 1.2332342863082886, -0.05183086916804314, -0.22485943138599396, 0.831416130065918, 1.9108394384384155, -0.8085787296295166, -1.5597418546676636, 1.2411409616470337, -0.7042360305786133, 0.5009320378303528, -1.0802503824234009, 0.5784285068511963, -0.4671440124511719, 0.9002493619918823, -0.7876971364021301, 0.6658058166503906, 0.2604207694530487, -0.6095471382141113, -0.23867793381214142, -1.5515302419662476, 1.7713489532470703, 0.17238906025886536, -1.9036365747451782, 0.6279038786888123, -2.3993124961853027, -1.3167394399642944, 0.5757833123207092, 1.0836818218231201, 0.21319229900836945, -0.7733177542686462, 0.4181009531021118, 0.9144922494888306, -0.34190624952316284, -0.9058122038841248, 0.5234060883522034, -0.26726993918418884, -0.65550297498703, -0.43582332134246826, -1.8038170337677002, -0.050222184509038925, -1.9281678199768066, 0.5071417689323425, -1.0329620838165283, -0.27290886640548706, -0.7496196031570435, -0.948323130607605, 0.351288765668869, 0.39388781785964966, 0.6577332019805908, -0.6622164249420166, 1.5166791677474976, -0.04793669283390045, -0.027072688564658165, 0.4490283131599426, -0.11833763122558594, -0.6511051654815674, 1.5550588369369507, 0.9673696160316467, 0.745314359664917, -2.033691644668579, 0.25884321331977844, 0.71192467212677, -0.22104892134666443, -2.0914623737335205, -0.6676270365715027, 0.14032600820064545, -1.2978808879852295, -2.8957529067993164, -0.26426345109939575, -0.18819892406463623, 1.1758854389190674, -0.6133952736854553, -1.3403767347335815, -0.18888433277606964, 1.1060677766799927, 0.002659468911588192, 1.0760074853897095, -0.2461884766817093, 1.0484367609024048, -0.05656522884964943, 0.5097987651824951, -0.5714720487594604, -1.9403458833694458, -1.6080490350723267, -1.1172513961791992, 0.14509496092796326, 0.196913480758667, -1.6546357870101929, 0.3180813789367676, -0.49701398611068726, 0.016535846516489983, 1.5963387489318848, 0.6909794807434082, 0.48499003052711487, -0.5911821126937866], [-1.2864302396774292, -0.9406098127365112, -1.4081937074661255, -0.4225725829601288, -0.16037523746490479, -0.7642636895179749, -0.6086977124214172, 0.037056922912597656, -0.5394694805145264, 0.5111126899719238, -0.6071351170539856, 0.5707187652587891, 0.02482716366648674, -1.4478923082351685, 0.9467079043388367, -0.48045361042022705, -0.8666024804115295, -1.4573209285736084, -0.31104227900505066, 1.0270287990570068, 1.311374545097351, 1.4846956729888916, -0.5016567707061768, -0.04131642356514931, -0.05159076675772667, -2.049779176712036, 0.4856836497783661, 0.2698514461517334, 0.3679162561893463, 0.0959504172205925, 0.7118039727210999, 0.045020926743745804, 0.34949588775634766, 0.8595196604728699, 0.49537548422813416, -0.340729683637619, 1.2497680187225342, 0.5740764141082764, 0.7488643527030945, -0.02777576632797718, 0.2959607243537903, -2.903385639190674, 0.44045716524124146, 1.4665559530258179, 0.16026023030281067, 0.8386685848236084, 0.7827854752540588, -1.358036994934082, -0.49160680174827576, 0.05529213324189186, 0.17847943305969238, 0.19826090335845947, 0.6071869134902954, 0.5410505533218384, -0.5733139514923096, -1.4977461099624634, 0.18526414036750793, -1.0083014965057373, 1.0161415338516235, 0.7687555551528931, 0.8737677931785583, 1.598211646080017, -0.11598518490791321, 0.6691564321517944, -0.8937496542930603, -0.31076228618621826, -0.4050871431827545, 0.19113120436668396, 1.1056156158447266, 0.04369061440229416, 1.5515583753585815, 1.0592817068099976, -0.4209462106227875, 0.28346461057662964, -0.6227483153343201, -1.311898112297058, 1.7294862270355225, 1.3130630254745483, -0.2389722764492035, 0.8901301026344299, 0.1724328100681305, 0.5946865081787109, -0.4193553924560547, -0.14908570051193237, -1.2673214673995972, 0.17460300028324127, 1.3396002054214478, 0.3364273011684418, 0.5155758261680603, 2.190211534500122, -0.4339239299297333, 0.15649394690990448, 0.08952793478965759, -1.054916501045227, -0.3817404508590698, -0.6608672738075256, -0.5820518732070923, -2.7778918743133545, 0.9372061491012573, -0.6124703288078308, 0.7488695383071899, 2.1304523944854736, -0.37112951278686523, 0.730506420135498, 0.8146321773529053, 0.9533789753913879, -1.1133878231048584, 1.4493108987808228, -0.013292200863361359, -0.5967431664466858, -0.4696943759918213, 0.12659424543380737, 1.0240685939788818, 0.7103468179702759, -1.2803921699523926, 0.583183228969574, -0.3200801610946655, 0.9025595784187317, 0.9498392939567566, 1.4992938041687012, -0.783802330493927, -0.4266643822193146, -0.2961026430130005, -0.28030166029930115, 0.8545295596122742, -0.13517065346240997, 1.036967396736145, -0.4602167308330536], [0.8000430464744568, 0.27760905027389526, 1.508948564529419, -0.4119258224964142, 0.7037650346755981, 0.09798896312713623, 1.2264440059661865, 0.15942437946796417, 0.6580092906951904, 0.7708291411399841, 2.061475992202759, -0.31184518337249756, 0.4572663903236389, -0.18997624516487122, 0.5182398557662964, 0.29010888934135437, 1.0494208335876465, -0.3734950125217438, -0.13723042607307434, -2.380080223083496, 0.4618620276451111, 0.1940508633852005, 1.8496142625808716, 0.16696682572364807, 0.648853600025177, -0.21553993225097656, -1.8371264934539795, -0.6659364700317383, 0.8978442549705505, 0.1576402485370636, -1.0758531093597412, -2.096011161804199, -0.017565740272402763, -0.8067179322242737, 1.043199062347412, 0.053477492183446884, -0.5813193917274475, 0.5092154741287231, -0.8421168327331543, 1.1424368619918823, -0.3395845592021942, 0.18121880292892456, -1.3771703243255615, 1.2059733867645264, 3.4698381423950195, 0.8579230308532715, -0.7624855041503906, 0.5087942481040955, 0.4918537735939026, 1.3123326301574707, 1.111075520515442, -0.9948127269744873, 1.2096792459487915, -1.2651475667953491, 0.1160687506198883, 1.2779031991958618, -1.0948984622955322, 0.2652271091938019, -1.6708918809890747, -0.7454444169998169, 1.1384408473968506, 0.8083745241165161, -0.6399542093276978, 0.6185500025749207, -0.41176852583885193, 0.23130109906196594, -0.4069705903530121, 0.6400631070137024, -1.917864441871643, -1.4939852952957153, -0.6931261420249939, 0.4643881618976593, -1.5124554634094238, 0.7462847232818604, 1.8415008783340454, 0.32364794611930847, 0.07564910501241684, -0.7643004655838013, 0.33709052205085754, -0.12622854113578796, 1.8746308088302612, 2.454277276992798, 0.41219794750213623, 0.4580122232437134, 0.6732673645019531, -0.5325409173965454, -1.5613322257995605, 1.0423978567123413, -0.8970410227775574, 0.06962364912033081, 0.3006797134876251, 0.4430079162120819, 0.2709527909755707, -0.0138546172529459, 0.4316141903400421, -0.40161287784576416, 0.2370879352092743, -0.3876463770866394, 1.9110138416290283, 1.7360270023345947, -0.21585038304328918, 0.14967699348926544, 0.021703410893678665, -0.8693311810493469, -0.3485683500766754, -0.319022536277771, -0.5443957448005676, 1.5563184022903442, -0.0700889378786087, 0.37668803334236145, 2.132654905319214, 0.3249224126338959, 1.7948237657546997, -0.6841837763786316, -0.7516117095947266, 0.2384064793586731, -0.8069108724594116, 0.3553768992424011, -0.69162917137146, -0.6341233253479004, 0.3641471862792969, -0.29559528827667236, -0.7998982071876526, -1.3764009475708008, -1.2838844060897827, -0.6596519351005554, 1.9768046140670776, 0.1640027016401291], [-0.03680144622921944, 1.0127612352371216, 1.826758861541748, 0.20749643445014954, 1.4535082578659058, -0.28950396180152893, 1.1633051633834839, -0.08226149529218674, 1.4521468877792358, 0.2616744041442871, -0.1520678699016571, 1.4680125713348389, -0.4903222322463989, -0.05612756311893463, 0.6609936952590942, 1.4089996814727783, -0.3087816834449768, 0.891100287437439, 0.049567848443984985, -1.0178263187408447, -0.6602705717086792, 0.08753082901239395, 1.4768824577331543, 1.1653919219970703, -0.3293251097202301, -0.72516930103302, 2.2402822971343994, 1.170725703239441, -1.3065012693405151, -0.011027291417121887, 1.9080989360809326, 1.5802031755447388, -1.1785614490509033, -2.5467238426208496, 0.9235764145851135, -1.294083833694458, -0.8318848013877869, 0.35246509313583374, 1.759677529335022, 1.0793488025665283, -0.11365730315446854, 0.5296629667282104, -1.0109020471572876, -0.4591560959815979, 0.5893855690956116, 0.03294515982270241, 0.5216041803359985, -0.594308078289032, 1.4723231792449951, 0.2171829640865326, 1.5039443969726562, 2.7655398845672607, -0.8244000673294067, 0.28543350100517273, -0.3678343892097473, -0.3366933763027191, -0.10773767530918121, 0.5841479301452637, 0.04500773921608925, 0.07286377996206284, 0.5927274823188782, -1.3265494108200073, 0.04044903442263603, -0.22041063010692596, -1.8741153478622437, 0.6116737723350525, 0.47209444642066956, 1.180503487586975, -0.20428383350372314, -1.4836490154266357, -0.4796389639377594, 0.8638951778411865, 0.45855921506881714, -0.2712059020996094, 0.961409866809845, 0.14289548993110657, -1.541872262954712, -2.0131444931030273, 0.28858762979507446, -0.44901418685913086, -0.1740923523902893, 1.085089087486267, 1.431462287902832, 1.8570940494537354, -0.5490501523017883, -1.4896866083145142, -1.8135955333709717, 0.12615616619586945, -2.084981918334961, 0.2274060696363449, -1.048155665397644, -0.34074679017066956, 0.4626527428627014, -1.4233975410461426, -0.2467958778142929, -0.7340630888938904, -0.9955815672874451, 0.7685922384262085, -0.5253556966781616, -0.230552539229393, 0.1745767742395401, 1.7160793542861938, 0.7581731677055359, 1.092392921447754, 0.39744794368743896, 0.3446493148803711, 0.8909921646118164, -1.7649637460708618, 0.4972265064716339, 0.5704648494720459, -0.14481571316719055, -0.8576288819313049, 0.4675464332103729, -1.7551194429397583, 1.6096606254577637, 0.9712414741516113, -0.5374245643615723, -0.16471995413303375, -0.4631302058696747, 1.3128551244735718, 1.1917487382888794, -0.33345481753349304, 0.24109824001789093, -0.2972831428050995, 0.41933175921440125, -0.08420398831367493, -0.4432164132595062, 0.9568894505500793], [0.1044735461473465, -1.1887294054031372, -0.29320842027664185, -0.9873883128166199, -1.3531900644302368, 0.16353081166744232, 0.4551786482334137, 0.16102391481399536, -1.2501219511032104, 1.1046169996261597, 0.9316409230232239, -1.760475754737854, 0.23921947181224823, 0.7302502393722534, 0.5314520001411438, 1.9371851682662964, 1.2952324151992798, -1.0635885000228882, 0.6193556785583496, -2.35015869140625, -0.2919919788837433, 0.6921700835227966, -1.1822340488433838, 0.19340723752975464, 0.1298888921737671, -2.208752393722534, -0.10845772176980972, -0.5176720023155212, 0.6628892421722412, 1.009392499923706, -0.7938235402107239, 1.0617929697036743, -0.31182152032852173, -1.4618020057678223, -0.2517050802707672, 0.1976802498102188, 1.280371904373169, 0.3465060889720917, 1.2619378566741943, -0.9957333207130432, 0.9208282828330994, -0.6428104043006897, 1.3372571468353271, -1.4581528902053833, 0.45714718103408813, 0.24817989766597748, -0.4555528163909912, 1.3396669626235962, 1.041678786277771, 0.2599967420101166, -0.10593234747648239, 1.819183111190796, -0.5428086519241333, 0.9346213936805725, 0.11561909317970276, -0.06490075588226318, 0.21812336146831512, -0.5983142852783203, -1.2687698602676392, 0.46883612871170044, 1.057028889656067, 0.43933799862861633, 1.0390357971191406, 1.5962361097335815, 1.3887277841567993, -0.037384845316410065, 0.4432354271411896, 0.22928182780742645, -1.3626213073730469, 0.9377244710922241, 1.311203122138977, 2.5455117225646973, 0.1919122189283371, 0.31976839900016785, 0.21122565865516663, 1.871048092842102, 0.046977538615465164, -1.505743145942688, 0.09408116340637207, 0.3699873387813568, 0.36842504143714905, -0.6800699830055237, 1.3835086822509766, -1.9397902488708496, 0.0386895090341568, -1.982666254043579, 0.7983017563819885, 0.08301515132188797, 1.8298112154006958, -0.22654391825199127, -0.5705928802490234, 0.20068003237247467, 0.5407873392105103, 1.4317216873168945, 2.240668773651123, 0.4356084167957306, -1.0807135105133057, 0.006371916737407446, 1.1122974157333374, 0.943434476852417, 1.4361451864242554, 1.0145899057388306, 0.560133695602417, 0.5215003490447998, 0.6633070111274719, -0.8972636461257935, 1.6029764413833618, -0.5453208684921265, -0.5030078291893005, -0.04998871684074402, -0.7124826908111572, -0.11002087593078613, -1.0250765085220337, 0.1859419345855713, 0.208116814494133, -0.4527830481529236, 0.7626046538352966, 0.42417770624160767, -0.5916182994842529, -1.221064567565918, 0.9166616797447205, -0.05242885276675224, -0.7294141054153442, -0.4672509431838989, 0.6925830245018005, 1.1179540157318115, -0.08518385142087936, -1.0348588228225708], [-0.2686121165752411, -0.7795184254646301, 1.366607666015625, 0.735192060470581, -2.0275652408599854, -0.3016469180583954, 0.77790766954422, 0.16985034942626953, -0.15934157371520996, 1.029398798942566, 0.6653238534927368, -0.16824547946453094, -0.2464239001274109, -0.5407261252403259, -0.19145239889621735, -0.003732770448550582, -0.1263531893491745, -1.2026792764663696, 0.4514175057411194, 0.8203341364860535, -0.10054576396942139, -0.339758038520813, -1.417688012123108, -1.0492233037948608, -0.8674338459968567, -0.7352785468101501, -0.8645598888397217, -0.6553235650062561, -0.13031332194805145, -2.230103015899658, -0.02294595167040825, -1.2632603645324707, -0.1566299945116043, -1.5771453380584717, -1.2240960597991943, 1.4409959316253662, 0.44941484928131104, -0.9354802966117859, 0.020193321630358696, 0.4346346855163574, 1.0501127243041992, -0.07445865124464035, 1.0081154108047485, -0.2716182768344879, -0.5034066438674927, -0.7941250801086426, 0.9245307445526123, 0.15846110880374908, -1.647341251373291, 1.4726066589355469, 0.054237473756074905, 1.6938472986221313, -0.24230380356311798, 0.705544650554657, -0.7971477508544922, -0.5928152799606323, -0.2944321036338806, -0.07767564803361893, 1.392551064491272, 0.8426432013511658, -1.08740234375, 0.4710662364959717, 0.27378755807876587, 0.47632357478141785, -1.591805100440979, -0.4525567889213562, -0.1909802109003067, -0.3094368278980255, 0.6815810799598694, 0.8255165815353394, -0.14425227046012878, 0.4701383709907532, 1.3360586166381836, -0.3847719430923462, -0.39122194051742554, -0.12580150365829468, -0.46242743730545044, 1.795100450515747, -0.9228523373603821, -0.757012665271759, 0.9441424608230591, 1.0982805490493774, 1.3473237752914429, -0.942895770072937, -0.8716007471084595, -0.16464826464653015, -0.9507518410682678, -0.4664331376552582, 0.47486093640327454, -1.617594599723816, 1.4233372211456299, 0.015405837446451187, -1.7821109294891357, 1.477927803993225, -1.2034039497375488, -0.510844886302948, -0.5007436275482178, 0.045571424067020416, -0.32280582189559937, 1.2640399932861328, -1.007485270500183, 0.5183030366897583, 0.3972185254096985, -0.009841457940638065, 1.1390104293823242, 0.36660680174827576, -0.22492846846580505, -0.11278428137302399, 0.9900575280189514, 2.85477352142334, -1.045589566230774, -0.4327608048915863, 0.33710378408432007, -0.6072806715965271, -1.6655645370483398, 0.9432735443115234, 1.087454080581665, 0.8800973296165466, 0.37921416759490967, -0.9439065456390381, -0.4090310335159302, -1.306974172592163, -1.4607810974121094, 1.969543218612671, 1.7173291444778442, -0.5242658853530884, -1.397063136100769, 0.10810552537441254], [0.2166830450296402, 0.7852283716201782, -0.028223440051078796, 0.2312713861465454, 0.21816352009773254, 2.8058056831359863, 0.6657280921936035, 0.4672398865222931, 0.15358039736747742, 1.2671263217926025, 0.2083740532398224, -1.1231400966644287, -1.653609275817871, -0.9142298102378845, 0.6334742307662964, -0.7195965647697449, -0.46102970838546753, -0.6589841246604919, -0.8927674889564514, 0.35748037695884705, -0.4490991234779358, 1.5541324615478516, -1.711318850517273, -0.04357537627220154, 0.8636502623558044, -1.0068353414535522, -1.5978186130523682, 1.7446905374526978, 0.37426263093948364, 0.7453216910362244, 0.017948660999536514, 1.0572699308395386, -0.25745806097984314, -1.5745564699172974, 1.7805346250534058, 1.0047811269760132, -0.06884465366601944, 0.9985597133636475, 1.1611826419830322, -0.6509393453598022, -0.8714408278465271, -0.4801764190196991, -1.775210976600647, -0.48133495450019836, -1.4429727792739868, -0.49759137630462646, 0.8325718641281128, 0.8260226249694824, -1.010166883468628, -1.464241862297058, -0.37101849913597107, -0.3581041395664215, 0.13067412376403809, -2.148422956466675, -0.5053144097328186, 2.3465535640716553, 1.6601086854934692, 1.0358082056045532, 0.15019434690475464, -0.9120731949806213, -0.030288297683000565, 0.9536609053611755, -1.370017409324646, 0.5486972332000732, 0.34368789196014404, -1.1450507640838623, 0.11422330141067505, 0.09399097412824631, -0.09323009848594666, -1.4439632892608643, -0.36108195781707764, -0.2100965827703476, -0.4469265639781952, -0.7980807423591614, 0.5338172316551208, -2.217121124267578, 0.48634886741638184, -0.47007474303245544, 1.362432837486267, 1.6873602867126465, 0.19423413276672363, -0.18987305462360382, 0.6279640793800354, -0.6621710062026978, -0.6411495804786682, 0.5911665558815002, 0.21644842624664307, 1.04438316822052, 0.5590285062789917, 0.3952482342720032, -1.6319024562835693, 1.0738179683685303, 0.7877583503723145, 1.7400683164596558, -0.5446556210517883, 0.361853688955307, 0.9014566540718079, -0.7308380603790283, 1.871389389038086, -1.3961693048477173, 0.7260664701461792, 0.8357518911361694, -0.8575359582901001, -0.25747841596603394, -0.40005314350128174, 0.6964988708496094, 1.4556437730789185, -1.554720163345337, 1.1948168277740479, 1.4597631692886353, -1.32685124874115, 0.6211540102958679, -1.4868375062942505, 0.9727198481559753, 1.979323148727417, 0.9039032459259033, -1.1162716150283813, -1.6475030183792114, -0.5008391737937927, 0.12761688232421875, -1.060876488685608, -0.40129464864730835, 0.7690796852111816, -0.4720304310321808, -2.8624167442321777, 0.5423724055290222, -1.098728895187378, -0.13106995820999146], [-1.7919062376022339, -0.5469270348548889, -1.4449396133422852, 0.015298665501177311, 1.0025906562805176, 0.5503202676773071, 0.5328980088233948, 1.0890792608261108, 0.17375294864177704, 0.9968735575675964, 1.3992455005645752, -1.0323798656463623, -0.09257661551237106, 0.38347673416137695, -1.5154354572296143, 0.5136014223098755, 0.2282601147890091, 0.574799656867981, -2.375437021255493, 0.4488210678100586, 0.14259524643421173, -0.9807693362236023, 0.17440113425254822, 1.970819354057312, 0.15677912533283234, -0.329256147146225, 0.9517865180969238, -0.30666297674179077, -3.1389963626861572, -0.08712553977966309, -0.21820484101772308, 1.315581202507019, 1.1650251150131226, 0.24153392016887665, 0.6998282074928284, 0.04242662712931633, -0.125960573554039, -2.379692792892456, 0.02777414582669735, -0.29734504222869873, -0.3614511787891388, -0.20444844663143158, -0.42998620867729187, 1.4472728967666626, 0.3039562404155731, -2.271684169769287, -1.3051273822784424, -1.3373970985412598, -0.325954407453537, 0.06604789942502975, -0.41696009039878845, -0.6183850765228271, -0.21159963309764862, 0.6080316305160522, -0.808502733707428, 1.3286620378494263, -1.3589104413986206, 0.9918205738067627, -0.9798195958137512, 0.012839125469326973, -1.1938401460647583, -1.4946775436401367, -0.9309188723564148, 0.16570572555065155, -1.4601707458496094, -0.9709855318069458, -0.8101339936256409, -0.139214426279068, -0.8380381464958191, 0.42141610383987427, 0.8922685980796814, -0.22539974749088287, 0.7815411686897278, 0.34229499101638794, 1.1109108924865723, 0.3662642538547516, 0.7147271037101746, 0.4402858018875122, 0.9637870192527771, -0.5057390332221985, -0.20247980952262878, -1.033969759941101, -0.21989397704601288, 0.4507353901863098, -0.254706472158432, 0.169633150100708, -1.2044627666473389, -0.47166723012924194, -0.08874329924583435, 0.19864152371883392, -1.039511799812317, -0.10299371182918549, -1.018547773361206, 0.7192819118499756, -0.4369468688964844, -1.3792635202407837, -1.5569406747817993, -0.2695971131324768, 0.23428666591644287, -0.15985332429409027, -1.3738458156585693, -1.7443283796310425, -0.7403014302253723, -0.7409083843231201, -0.24065586924552917, 0.6912752985954285, -2.5626862049102783, -0.4697657823562622, -0.8200021386146545, 0.48987266421318054, 0.37036848068237305, -0.7122719287872314, -0.30992019176483154, 1.1256321668624878, 1.529478669166565, 0.1870279163122177, 0.13414455950260162, 0.19706055521965027, 1.2562767267227173, -0.37913838028907776, 4.358133792877197, 0.14161324501037598, -0.6868207454681396, 2.2908153533935547, -0.26267027854919434, 0.5162164568901062, -0.40625452995300293, -1.913337230682373], [-1.0117104053497314, 0.9335201382637024, -0.451303094625473, -1.2263296842575073, 0.7632288336753845, -0.8278417587280273, 0.010565963573753834, 1.6200591325759888, 0.6582101583480835, -0.30094197392463684, -0.04913479834794998, -1.323626160621643, 0.26711031794548035, -1.0938215255737305, -0.44224056601524353, -0.7307794094085693, -0.11868131160736084, -0.25117847323417664, -0.9532260298728943, 1.4012197256088257, -1.367928385734558, 0.46631526947021484, -0.11269596219062805, 0.7227606773376465, 0.5273635387420654, -0.3858319818973541, 0.07628729939460754, -0.3612438142299652, -0.5287783145904541, -0.20330001413822174, -1.3272496461868286, 0.4545709490776062, -0.8892456889152527, 0.1719239205121994, -1.7149739265441895, 0.3349698781967163, 0.6627826690673828, 0.5683130025863647, 0.759637176990509, -0.13745249807834625, -0.03627949580550194, 0.16453932225704193, -0.4416479766368866, -1.8035930395126343, 1.197482705116272, -0.29650524258613586, -0.28766897320747375, 0.1917944699525833, -0.7285622358322144, 0.7734291553497314, 0.2673848271369934, 0.37748003005981445, -0.4371030926704407, 0.018593881279230118, -0.7673671245574951, 1.0099951028823853, -0.28341802954673767, -1.6518536806106567, -0.2695118486881256, 0.6850807070732117, -0.0049775647930800915, -0.3721587061882019, -2.0886502265930176, -2.98928165435791, 0.590541660785675, -0.45414769649505615, 0.39861392974853516, 3.171799421310425, 1.198190450668335, 1.2567176818847656, 0.8465509414672852, -0.6947256922721863, 0.2733912467956543, 0.41775596141815186, -0.37389782071113586, -0.48544371128082275, 0.08516072481870651, -0.2691684067249298, -0.012329933233559132, -0.0838053748011589, 0.9727767109870911, -0.9108471274375916, 0.32201796770095825, -1.2192022800445557, 0.7231977581977844, 0.05641915649175644, -1.112402319908142, -0.7803781032562256, -0.4110848307609558, 1.0128408670425415, 0.7528330087661743, -0.40997061133384705, 0.946800708770752, -0.27734634280204773, -1.2780686616897583, -1.1184232234954834, 0.7246096134185791, -0.9172729253768921, 0.01093009952455759, 0.6286709308624268, 0.15824581682682037, 1.9329240322113037, -0.5247431993484497, 1.7385588884353638, 0.5013198256492615, 1.6691179275512695, -1.0350841283798218, 0.10984718054533005, 0.8824743628501892, 0.5146358609199524, -0.64913010597229, -0.34961625933647156, 0.2713601887226105, -1.4872784614562988, -1.5476309061050415, -0.35032886266708374, 0.5765484571456909, 1.4570873975753784, -0.35373663902282715, -0.2548862397670746, -2.790790557861328, -0.5438397526741028, -0.03623490035533905, -1.1118738651275635, -1.942761778831482, -0.5037588477134705, 0.9223284125328064, 0.30002522468566895], [0.8466194868087769, 1.837976336479187, -1.0936403274536133, 1.186095118522644, -0.43726298213005066, -0.2130466252565384, 0.8238608837127686, 0.24688006937503815, -0.7064855694770813, 0.18285442888736725, -0.6027823686599731, -0.04057598114013672, -1.337611436843872, 0.15097588300704956, 0.4274248480796814, -0.45982998609542847, 0.10836411267518997, 1.959058165550232, 0.7264293432235718, 0.23089636862277985, -1.4490660429000854, 1.2078914642333984, -0.8091011643409729, -0.9463975429534912, 1.1813623905181885, 0.5295292735099792, 1.2962137460708618, 1.2003616094589233, 0.712067723274231, -0.6832323670387268, -0.3386135697364807, -0.34866002202033997, -0.6478906273841858, 0.86167973279953, 0.9409284591674805, 0.9189414978027344, 1.479127049446106, 1.423805832862854, 0.40872707962989807, 0.9446740746498108, -1.0025047063827515, 0.8312131762504578, 0.4130174517631531, 2.0538899898529053, -0.2817881107330322, -0.7700937986373901, 0.09497398883104324, 2.0595288276672363, -0.0946805328130722, -0.2824881672859192, 1.491692304611206, 1.0811902284622192, -0.14841504395008087, 0.8633678555488586, -0.6912017464637756, -1.4573029279708862, -0.31533703207969666, -0.7238425016403198, -1.5727020502090454, -0.904904842376709, -0.29384908080101013, -1.0499144792556763, 1.994815468788147, 1.0931352376937866, 0.037446264177560806, -0.19878822565078735, 0.04901763051748276, 1.5153741836547852, 1.0528361797332764, -1.2926771640777588, -0.7853744626045227, -0.7618537545204163, 0.6468906402587891, 0.2838287949562073, 0.0732516497373581, -0.43803641200065613, 1.304041862487793, -1.5466631650924683, -1.0884100198745728, -1.3938994407653809, -0.6544229388237, 0.46389347314834595, 0.03734970465302467, 0.8913193941116333, -1.2364615201950073, 0.5126781463623047, -0.07636549323797226, 1.819471836090088, 0.2580806612968445, -1.2232513427734375, 1.3963661193847656, 0.07148782908916473, -0.5632786154747009, -0.09295166283845901, 0.3826117515563965, 0.27891266345977783, 1.100400686264038, -0.1516876071691513, 0.9120253324508667, 1.1342058181762695, -0.762453019618988, 1.610018014907837, -1.145342469215393, -0.91465824842453, -0.5562298893928528, -0.15968072414398193, -1.0607752799987793, 0.7296485304832458, -1.1046181917190552, 0.8085893988609314, -0.6672510504722595, -0.7173900008201599, 0.23976659774780273, -1.6041277647018433, 0.8752736449241638, -0.8469345569610596, -2.6540095806121826, 1.5374470949172974, 1.742480754852295, 0.5955769419670105, 0.07814965397119522, 0.1025855615735054, 2.094060182571411, 0.13152393698692322, -0.8668789267539978, -0.21166178584098816, -0.5998049974441528, 0.4206589460372925], [0.2856854498386383, 0.05071140453219414, -0.6887101531028748, -2.1497385501861572, 1.3408854007720947, -1.181209683418274, -0.7258595824241638, 1.6963249444961548, 1.4162616729736328, 0.6308690905570984, -1.088204264640808, 0.7456018328666687, 0.4510505795478821, 0.5370715856552124, -0.8429857492446899, 0.4912719130516052, 1.8146001100540161, 0.3583469092845917, -1.217572569847107, -0.15162639319896698, -0.07977811247110367, -0.714358389377594, 0.7699984908103943, -0.8419811129570007, 0.1351512372493744, 0.3516247272491455, -0.8753753900527954, -0.16500699520111084, -0.8778193593025208, 0.9296826124191284, -1.1119939088821411, -0.9045670032501221, 1.2776353359222412, -0.37852343916893005, -1.5635076761245728, -1.392924189567566, -1.524101734161377, -1.6344385147094727, 0.33705630898475647, 0.29792097210884094, -0.03453169763088226, 0.8286817669868469, 0.6723463535308838, 1.3110742568969727, -0.44420742988586426, -0.38614287972450256, -0.9955267906188965, 0.15872594714164734, 1.062048077583313, -0.8140767216682434, -1.6508748531341553, -0.35159918665885925, 2.3982763290405273, 0.012807314284145832, 0.991329550743103, -1.40389084815979, -0.4310337007045746, -0.7273237705230713, 1.4068289995193481, -0.01686880551278591, 0.4487609267234802, 0.6235225200653076, 0.6730425953865051, 2.462475299835205, 0.06664825975894928, 1.4102622270584106, -0.38699668645858765, -0.5022294521331787, 1.310518741607666, -0.36935240030288696, 2.3903093338012695, 0.41021302342414856, 1.0178602933883667, 0.6731606125831604, 0.47434887290000916, -1.1069817543029785, 0.5594871640205383, -0.22823002934455872, 0.5266968011856079, 0.29911133646965027, -0.44388049840927124, 1.1183544397354126, -0.7043244242668152, -0.6486608386039734, 0.6506369113922119, -1.0473737716674805, 0.8643016815185547, -0.5531091094017029, 0.053894154727458954, 0.8734946846961975, -0.332968533039093, 0.7437638640403748, 1.592299461364746, 0.6927591562271118, 0.7656265497207642, 0.798627495765686, 0.9438679218292236, 0.026805082336068153, 0.1415417641401291, 0.1929139792919159, -0.32588180899620056, -0.24001848697662354, 1.0471343994140625, 2.3487343788146973, -0.02717159129679203, -0.9078903198242188, 0.26602447032928467, 0.6298001408576965, 0.7203779816627502, 1.04121994972229, 1.5362221002578735, 0.297800749540329, 0.6785626411437988, -1.0362954139709473, 0.1119743287563324, -0.9762356877326965, 0.2894684374332428, 0.7582279443740845, -0.5742383599281311, 1.0886303186416626, -0.35830116271972656, 0.8277290463447571, -0.32417601346969604, -1.6489723920822144, -0.10329452157020569, -0.058557238429784775, 0.5365780591964722, 1.0164754390716553], [-0.07797982543706894, -0.41127800941467285, -1.5129903554916382, -0.9888431429862976, -0.2883751690387726, 1.8545739650726318, -1.4564180374145508, -0.556085467338562, -1.4943184852600098, -0.0195546243339777, 0.7546696066856384, 0.782127320766449, -1.2342026233673096, -1.218906044960022, 3.0307869911193848, 0.7679125070571899, 0.28479933738708496, -1.2959892749786377, 1.4078463315963745, -0.5348013639450073, 1.0142673254013062, 1.3335999250411987, 1.4938855171203613, 0.1626851111650467, -0.2741263210773468, 2.403485059738159, -2.3079659938812256, -1.0379018783569336, 0.4493095278739929, -0.28762125968933105, 0.6085277199745178, 1.1704086065292358, -0.6060412526130676, 1.2252404689788818, -0.922468364238739, 0.5090239644050598, 0.830793023109436, -1.5208985805511475, 1.2523770332336426, -1.661368489265442, -0.1743556708097458, 0.49141764640808105, 0.08879641443490982, 0.5929961204528809, -0.5184958577156067, 1.4586265087127686, -0.3042157292366028, 0.14470431208610535, 0.09785294532775879, -0.4642152786254883, 2.457961082458496, -0.12101173400878906, 1.0778305530548096, -1.3861455917358398, -1.4464200735092163, -0.5358512997627258, 0.055431582033634186, -2.25765323638916, 1.2560864686965942, -0.25999101996421814, 0.5778257250785828, 0.30274373292922974, -0.42593157291412354, -1.4652413129806519, -0.04274553433060646, 0.36536964774131775, 0.812351405620575, 0.22951550781726837, -1.11231529712677, -1.7172681093215942, 1.8428438901901245, 0.621620774269104, -1.6643574237823486, -0.8913230895996094, -1.6921660900115967, -1.2343891859054565, 0.1931341290473938, 1.0167423486709595, -0.9966773390769958, -1.068220615386963, 0.17386575043201447, 0.20424099266529083, 0.0747576653957367, -0.16138023138046265, 0.8059946894645691, 2.3291032314300537, 0.45804449915885925, 0.38324642181396484, 2.017130136489868, -0.7105146050453186, -0.022341979667544365, 0.1187799721956253, -0.7394886016845703, 0.9935998916625977, -1.4895075559616089, 2.2439985275268555, 0.06183220073580742, -0.11257676780223846, 0.3340550363063812, -0.1793413907289505, 1.0334222316741943, -2.451991319656372, -1.3211302757263184, -0.9851635694503784, 0.4920543134212494, -0.269768625497818, -0.44381800293922424, 0.023036764934659004, 0.09482858330011368, -0.14400146901607513, -0.3412332534790039, 0.5030616521835327, -0.530767023563385, 0.20535734295845032, 1.0823063850402832, -1.2452865839004517, 0.38807058334350586, 0.6909646391868591, 1.2120299339294434, -0.5622862577438354, -0.5392406582832336, 1.746989130973816, -0.13685716688632965, 0.1927214413881302, 1.9493886232376099, 1.6548542976379395, -1.5157794952392578, -0.1861904114484787], [1.8853377103805542, -1.2012594938278198, 0.8261412382125854, -0.5282706618309021, 1.3711220026016235, -0.12958484888076782, 1.617324709892273, 0.09254787862300873, -0.34882041811943054, 0.5640183687210083, -1.3888200521469116, -1.2460464239120483, -0.4202330708503723, 0.2756313383579254, -2.2440760135650635, 0.33327752351760864, 0.3933502435684204, -0.6429456472396851, -0.49272021651268005, -0.2693130075931549, -0.9472631812095642, -0.2602522373199463, 0.6808265447616577, -1.601565957069397, -0.3149987459182739, -0.9245532155036926, 0.2590944766998291, 1.367648959159851, 0.8084326982498169, 0.061910081654787064, -0.925971269607544, 0.7461667656898499, 1.7397068738937378, 0.18773910403251648, 1.5723276138305664, 0.23367749154567719, -0.6170943975448608, 1.5796043872833252, -1.319560170173645, 0.30109450221061707, -0.18356989324092865, 2.071141242980957, 2.265784740447998, 0.4689510464668274, -0.7935059070587158, -0.4967247247695923, 0.24261732399463654, -2.032109022140503, 0.14600060880184174, 0.23083433508872986, -0.6572570204734802, 0.13915185630321503, 0.6022141575813293, 0.4153164029121399, -1.7650675773620605, 2.3068532943725586, 1.2740705013275146, 0.3331669867038727, -0.09908828139305115, 0.28446581959724426, -0.6993299722671509, -0.7570850253105164, 0.05732329189777374, -1.0516520738601685, -0.8478567004203796, -0.7824109792709351, 1.0541982650756836, -0.03311124071478844, 1.1613283157348633, 0.3628154695034027, -1.85875403881073, -0.0021003324072808027, -0.6662813425064087, -0.1521724909543991, -0.22112765908241272, 0.06945192813873291, -0.25237730145454407, 0.6120046973228455, -1.032866358757019, 0.24794518947601318, 2.423092842102051, 0.6024203300476074, 0.12509119510650635, -0.5076801180839539, -0.3827034533023834, -1.2746859788894653, 0.7164760828018188, -0.5368106961250305, -0.6432360410690308, 1.157813549041748, 1.2417980432510376, -0.8707187175750732, 0.8138786554336548, 0.12350344657897949, -0.3614998459815979, -0.31184157729148865, 0.7435952425003052, 0.5122065544128418, -0.2669404447078705, 0.8456189036369324, -1.179919958114624, -0.490043580532074, 0.3603043556213379, -1.3273394107818604, 0.26389285922050476, -0.45120081305503845, -0.6514290571212769, -0.3941807150840759, -0.5052582025527954, 0.5733169913291931, -2.2660038471221924, -1.843817114830017, 0.5208407044410706, -0.05011036992073059, 0.744415819644928, -1.3821971416473389, -0.8885790705680847, 1.031794548034668, 0.3571576178073883, -0.48883697390556335, 0.4463815689086914, 0.3476526439189911, -1.2772780656814575, 1.106459617614746, -2.268436908721924, 0.3639530539512634, 0.7990981936454773, -1.090627908706665], [1.8032236099243164, 0.2544737458229065, -0.3554230332374573, -0.13660620152950287, 0.5649319291114807, 0.5762772560119629, 0.7987431287765503, 0.978918731212616, 0.5276716351509094, -0.3982802629470825, 0.3398520052433014, -1.77336585521698, -0.740542471408844, -1.9305214881896973, -0.7912283539772034, -1.323599100112915, 0.22102630138397217, 0.8081075549125671, -0.32193315029144287, 0.6017892956733704, -0.35464373230934143, -0.6168015599250793, -0.42883527278900146, 1.318687915802002, -0.5116541981697083, 0.6520110368728638, -0.230460062623024, -0.5445789694786072, -1.0459831953048706, 0.41023752093315125, -0.6373578310012817, -0.45319831371307373, 1.1564987897872925, -1.2016935348510742, 1.905350685119629, 1.3646721839904785, 0.8701295852661133, 0.6534951329231262, -0.7161033153533936, 0.6854454278945923, -0.0661332830786705, -0.07840269058942795, 0.028128093108534813, -0.02944895066320896, 1.4433691501617432, 0.7118316292762756, -0.24862231314182281, -0.05085577443242073, 0.6527724266052246, -0.4333449602127075, 0.3726968765258789, 1.5528908967971802, 0.7644163370132446, 0.4632473587989807, 0.09763088822364807, -1.0551657676696777, 0.5031464695930481, -3.560127879609354e-05, -0.18617352843284607, -0.30236145853996277, -0.45831435918807983, -0.4253408908843994, 0.3734707832336426, 1.665500521659851, -0.6343549489974976, 0.7988545298576355, 1.4353004693984985, -0.7386101484298706, -0.25519731640815735, -0.9199848771095276, -1.7190594673156738, -1.149214267730713, 0.2948075234889984, 0.5340098738670349, -0.434153288602829, 0.7583251595497131, 1.333317518234253, 0.32266074419021606, 0.7350070476531982, -0.3693859279155731, 0.6511572599411011, -0.6937059760093689, 0.20120292901992798, 0.1215607225894928, -0.021326478570699692, -1.1709808111190796, 0.21275882422924042, 0.9297031760215759, 0.1884002834558487, 1.2935807704925537, -0.474338561296463, -0.543037474155426, -1.7184864282608032, 0.10769762843847275, -1.0129300355911255, -0.26574432849884033, 0.886603057384491, -0.566214382648468, -1.358778476715088, 1.5010639429092407, -1.3501514196395874, 0.9796149730682373, 0.26234862208366394, -0.6665881872177124, 0.3977722227573395, -1.1229785680770874, -0.9261993169784546, 1.050491213798523, 0.6243250370025635, 1.3504555225372314, 0.36016204953193665, -0.6819942593574524, -2.1080102920532227, -0.16448989510536194, -0.821456253528595, 0.5627307295799255, -0.6360304951667786, 0.052239157259464264, 0.29718393087387085, -1.3564459085464478, -0.16334058344364166, -1.456514835357666, -1.037514567375183, -1.2514504194259644, -1.3907479047775269, 0.4796012043952942, -1.8461180925369263, -0.3422764837741852], [-0.06271567940711975, -1.1163872480392456, -0.1831607222557068, 1.346483588218689, 0.5736571550369263, 0.6891177296638489, 0.2559172511100769, -0.38923946022987366, -0.974170982837677, -0.4997549057006836, 1.3079760074615479, 0.6324865221977234, -1.2355245351791382, -0.43159255385398865, 1.4694647789001465, -0.298939049243927, 0.08781932294368744, -0.3147990107536316, -1.041386604309082, 0.09414997696876526, -1.0663427114486694, 0.7323859930038452, 0.5868753790855408, 1.7042251825332642, -0.5508595705032349, 0.09560307115316391, -0.2949802875518799, -0.4342944025993347, -1.8340486288070679, -0.14197500050067902, 0.6948485374450684, 0.9057608842849731, 0.7608606219291687, 0.8557769656181335, 1.123290777206421, 1.0174022912979126, 2.4890711307525635, 0.6597754955291748, 0.12921947240829468, -1.474287986755371, 0.6630179286003113, -1.3689578771591187, -0.5136245489120483, 0.8832647204399109, 0.15023694932460785, 1.3060951232910156, -0.8889542818069458, 0.5624812841415405, 0.0592656210064888, 0.31723034381866455, 2.004396915435791, -0.6100723743438721, 0.1640237718820572, -0.15211567282676697, 1.577896237373352, 0.49808230996131897, 1.8104335069656372, 0.5751250982284546, 1.16327965259552, 0.05741716921329498, -1.2168844938278198, 0.5641921162605286, 0.8759986758232117, -0.061748430132865906, 1.3815069198608398, -0.583489716053009, 0.38631483912467957, -0.6007159352302551, -0.5592127442359924, 1.1532602310180664, 0.9389171600341797, 0.1660727709531784, 0.9593021273612976, -1.7546061277389526, 2.0706379413604736, -0.011696415022015572, 0.24676251411437988, 0.7799571752548218, -0.10064487904310226, 1.0334969758987427, 1.2049903869628906, -0.49452823400497437, 2.1654090881347656, 0.6228927373886108, 1.1592934131622314, -1.1062135696411133, 1.0658711194992065, -0.5255699157714844, -2.562786817550659, -1.106654405593872, 0.7316757440567017, 0.5690377354621887, 0.7806364297866821, -0.34933337569236755, -0.5076390504837036, -0.0019255526131018996, 2.0861055850982666, -0.010357958264648914, -1.102824330329895, -0.2688847780227661, 1.554669976234436, 0.6130382418632507, 0.9835019707679749, 0.33150774240493774, 0.5391654968261719, -1.2613252401351929, 0.12982463836669922, -1.4368772506713867, -1.9981777667999268, 0.14088349044322968, 1.5154242515563965, 0.7037099599838257, -1.2991172075271606, -0.10484316945075989, 0.21640026569366455, 0.25197356939315796, -1.1794742345809937, 1.2676805257797241, -0.8856030106544495, -0.4104304313659668, -0.4203778803348541, -0.19320493936538696, 0.10004378855228424, 0.4726814925670624, -0.08039907366037369, 1.0810492038726807, 0.4400179386138916, 0.6165766716003418], [0.19053184986114502, -1.5282450914382935, -0.8691707849502563, -0.060893088579177856, -0.46598511934280396, -1.729119062423706, -0.6626101732254028, 0.057087820023298264, 0.7409507036209106, -1.6860324144363403, 1.396989107131958, 0.1482090801000595, -0.8215794563293457, 0.3941073715686798, 0.15005356073379517, 0.5610260367393494, 0.140967458486557, -1.6846754550933838, -0.2253793179988861, 0.977229118347168, 0.17128288745880127, 0.9089693427085876, 0.7391437888145447, -0.9311903715133667, 0.9031960368156433, -1.1003199815750122, -0.22198668122291565, -0.9928163290023804, 0.686059296131134, 0.3122022747993469, 2.222355842590332, -0.9724791646003723, 0.7640063166618347, -1.0091437101364136, -0.6324438452720642, 0.7465107440948486, 0.3471725583076477, -0.9177529215812683, -0.9589341282844543, 0.36657780408859253, -0.7154597043991089, -1.6951165199279785, 0.7380507588386536, -0.19211791455745697, -0.2559460401535034, -0.6617609262466431, -0.4900042712688446, -1.0109153985977173, -0.9424991607666016, 1.0488413572311401, -0.986659824848175, 0.8097102046012878, -1.711771011352539, -1.0792210102081299, -1.0261448621749878, -0.31423312425613403, -2.7887346744537354, 0.13945460319519043, -0.8875839710235596, 0.39895761013031006, 0.1833258867263794, 1.7755067348480225, 1.8351610898971558, 0.08826104551553726, -1.3520234823226929, -0.4072423577308655, 0.717126727104187, -0.2469986379146576, -1.0887889862060547, -1.3424828052520752, 1.9345016479492188, -0.453138142824173, -0.1543489247560501, 0.800418496131897, -1.407444715499878, 0.765064537525177, -0.6055191159248352, 0.5161124467849731, -1.692460298538208, 0.2588675022125244, -1.424371600151062, -0.8004181385040283, 0.10208673775196075, 0.42871153354644775, 1.1008641719818115, -1.6808592081069946, -0.09659594297409058, 1.8280794620513916, 0.5533767938613892, -0.7574319243431091, 0.5493625402450562, 2.1859638690948486, -1.4193379878997803, 0.6161081790924072, 1.2295269966125488, 0.5447848439216614, -1.4148645401000977, -0.032009147107601166, 0.48559215664863586, -1.1055693626403809, 0.7905207872390747, -0.9612642526626587, 1.3483322858810425, -0.6575922966003418, -1.1703267097473145, 0.491110622882843, -0.5673192143440247, 0.4493824243545532, -0.39592212438583374, 2.3763792514801025, 0.9001085162162781, 2.073080062866211, 0.97716224193573, 1.0819566249847412, -0.2397335022687912, 0.3201316297054291, -1.048094391822815, 0.7327507734298706, 1.7091952562332153, 0.9711753726005554, -0.06260482966899872, -0.16366133093833923, 0.7738052606582642, 0.08655380457639694, -1.572179913520813, 0.1558765321969986, 1.5553911924362183, 0.09610423445701599]], [[1.1076993942260742, 0.8583053350448608, 0.5794647336006165, -1.2527809143066406, 0.4026085138320923, -1.4229276180267334, -0.34636110067367554, -0.8430273532867432, -1.1597431898117065, -0.31605422496795654, 0.1725936084985733, -0.24671751260757446, 0.31625890731811523, 0.31217122077941895, -0.08145055174827576, -1.0588116645812988, 0.5710375308990479, -1.4930206537246704, 1.8739879131317139, 0.3502015173435211, 0.967481791973114, -1.864306926727295, -0.9908199310302734, -0.3173833191394806, -0.4914827346801758, 0.11077670753002167, -0.8387874364852905, -0.06360411643981934, 0.8949213027954102, -1.0380957126617432, -0.10767029225826263, 0.33655351400375366, 0.05171402916312218, -0.38284775614738464, 1.2987092733383179, 0.001431593089364469, -0.14322611689567566, 0.8039087057113647, 0.1936969757080078, 0.35961198806762695, -0.09420507401227951, 0.7098488807678223, 0.29397034645080566, 0.5647801160812378, 1.460439920425415, -0.24801355600357056, 0.4886167645454407, 1.0803977251052856, 0.012935410253703594, 0.34311357140541077, 0.05694521591067314, -0.6005028486251831, 1.0895110368728638, -0.2586357593536377, 0.3309182822704315, -1.7608200311660767, 0.7039235830307007, 1.1619423627853394, 1.0525376796722412, -0.7555282115936279, 0.9559136033058167, 2.482985496520996, -0.3782656490802765, -1.1103850603103638, 0.4675942659378052, 1.2729032039642334, 1.7814109325408936, -0.08629753440618515, -0.17889462411403656, -0.19687095284461975, 1.3218995332717896, -0.005649943836033344, 0.13404889404773712, 0.1537645012140274, -0.5571644902229309, -0.7188653349876404, -1.3691434860229492, -0.19029299914836884, 0.457477867603302, -1.4721108675003052, -0.11658726632595062, 0.18967095017433167, 1.6607062816619873, -1.4543794393539429, 1.128082513809204, 0.8287872076034546, -0.17315952479839325, 1.571698784828186, 0.11953137069940567, -1.0422992706298828, -0.8422266840934753, 0.6488940119743347, 0.3863900601863861, -0.8625608682632446, -1.6431195735931396, 0.4396791160106659, -1.3927757740020752, 0.1774720400571823, -0.9711137413978577, -0.2751311957836151, 2.262110948562622, -0.6924116015434265, -0.3691087067127228, 1.1186050176620483, -1.3074150085449219, -1.3600627183914185, -0.4354305565357208, -1.1552821397781372, 1.2696994543075562, -0.35174912214279175, 1.0417189598083496, 0.5261865258216858, -1.0025807619094849, -0.5526770949363708, -0.9061224460601807, 1.4113495349884033, 0.2095409631729126, 1.164105772972107, -1.2338529825210571, 0.5321823954582214, 0.7276721596717834, 0.8723159432411194, -0.6483586430549622, 1.0112613439559937, -1.2942003011703491, 1.0939456224441528, -1.207323431968689, 0.13563179969787598], [-0.16871052980422974, -0.05263996869325638, 0.19606655836105347, -0.6884415149688721, 1.3887404203414917, 0.8824211359024048, -0.6584615707397461, -1.4192322492599487, -0.23576951026916504, -0.6522965431213379, -0.52947598695755, -0.5643247365951538, -0.10163000226020813, 2.618453025817871, 0.7079459428787231, 0.08761315792798996, 1.1055020093917847, -1.546373724937439, 0.015748314559459686, 0.7232307195663452, -2.2886061668395996, -0.291574090719223, -0.7873558402061462, 0.9287768006324768, 1.2724225521087646, -1.9086936712265015, -1.1666501760482788, 0.08716557919979095, -1.0855196714401245, 1.1891192197799683, 0.10827796906232834, -0.6945876479148865, -0.8662428855895996, 0.11001288890838623, -0.9427385926246643, -0.8835549354553223, 0.5471633076667786, -0.6478891372680664, -0.8342620134353638, -1.5186536312103271, -1.4396313428878784, -0.6288567185401917, -0.5532518625259399, -1.38884699344635, -0.6062284111976624, -0.8177880048751831, -0.776456356048584, -0.04640901833772659, -1.5437599420547485, -0.7225281000137329, 2.5670485496520996, 0.265249639749527, -1.951173186302185, -0.9379822015762329, -0.3867643475532532, 1.1933985948562622, -0.12483252584934235, 0.7496703267097473, -1.093528151512146, 0.21629799902439117, 0.6418988108634949, 0.27120745182037354, -0.857613742351532, 0.39132869243621826, -0.14023075997829437, -1.3963128328323364, -1.1513725519180298, -1.496110439300537, -1.979812741279602, -0.16039013862609863, 0.5695792436599731, 0.1545809656381607, 0.026796307414770126, 0.6016359925270081, 0.12712682783603668, 1.6938890218734741, 0.5815937519073486, 0.23111674189567566, 0.37974482774734497, 1.0022047758102417, -0.27451378107070923, -0.6383793354034424, -1.4665759801864624, -1.900620460510254, 0.5906904339790344, -1.1884162425994873, 0.6480895280838013, -1.4581400156021118, 0.0459052175283432, -0.4196663200855255, 1.1221083402633667, -1.8563299179077148, 0.334891140460968, -0.08987421542406082, 1.2416640520095825, 0.6455533504486084, 1.0947332382202148, -1.294456124305725, -1.8701503276824951, -1.0957722663879395, -0.44113415479660034, 1.2224541902542114, 0.21021762490272522, 0.639354407787323, 0.9732160568237305, -0.06041767820715904, -0.6600289940834045, 0.6832697987556458, -0.13443897664546967, 0.04128090292215347, 0.8423104286193848, -0.8770370483398438, -0.4246244728565216, 0.42674708366394043, 0.8916111588478088, 0.7429800629615784, 0.29025813937187195, 0.7979369759559631, 0.42109015583992004, -1.147235631942749, -0.2560918629169464, -1.4243131875991821, 0.6189197301864624, 0.21531249582767487, -0.6670136451721191, 0.6594564914703369, -0.17144149541854858, 0.3183498680591583], [0.27435705065727234, -0.674622654914856, 1.1469262838363647, -1.3757970333099365, 1.4986374378204346, 0.20061491429805756, -0.6275359392166138, 1.058993935585022, -0.6242765188217163, -0.025376930832862854, -2.359663486480713, 0.32761892676353455, -0.4765835404396057, 0.1495988816022873, -0.4596062898635864, 1.50657057762146, -0.6562535762786865, 1.0928959846496582, -0.4996755123138428, -0.9893050789833069, -0.5876345634460449, -0.46419739723205566, 1.0976775884628296, -0.669692873954773, 0.06403074413537979, 1.1336658000946045, 1.5818537473678589, 0.4097191095352173, -0.462310254573822, -1.5024546384811401, -1.5227742195129395, -0.022032927721738815, 1.3586220741271973, -0.9458777904510498, 0.9835790395736694, 0.4496850073337555, 0.25104305148124695, 0.2239234745502472, -0.3708147704601288, -0.9441196918487549, 0.8708248734474182, 0.20590239763259888, -1.9102357625961304, -0.7126598358154297, -0.4799984395503998, -0.300680935382843, 0.6306824088096619, 0.9196546673774719, -0.6124337911605835, -0.3784550130367279, -1.0718361139297485, 1.4912135601043701, -2.2247395515441895, -0.8879178166389465, 0.1924930065870285, 1.3965909481048584, -0.8972735404968262, -1.342063069343567, -0.060398925095796585, 0.19984491169452667, 1.0034747123718262, -0.7205259799957275, -0.7689046263694763, 1.2083543539047241, -0.16390569508075714, -0.8223657011985779, 0.9703625440597534, 0.7893911004066467, -0.13096106052398682, 0.005046376492828131, 0.04057568684220314, 1.5302345752716064, -1.0066579580307007, -0.08354990929365158, -1.1114778518676758, 1.051993727684021, -2.682988405227661, 0.2276683747768402, 0.876185417175293, 1.6959680318832397, 1.7423900365829468, -1.592134952545166, 1.1748179197311401, -2.0041754245758057, -0.9050306081771851, 1.3055776357650757, -0.7023662328720093, 0.0540083684027195, 0.5082893967628479, 0.5645213723182678, 0.6715699434280396, -0.06965656578540802, -0.6045283079147339, -0.08457092940807343, 0.5963194370269775, -0.3368077576160431, 0.12876909971237183, 1.631101131439209, -0.3543594479560852, 1.5112532377243042, 0.04849785938858986, 0.45850783586502075, 0.6106276512145996, 2.1633126735687256, -0.49872711300849915, -0.9745104908943176, -1.3514702320098877, -1.1329596042633057, 0.016314052045345306, 0.9695044159889221, -0.8067443370819092, -0.3019534647464752, -1.5323338508605957, -0.2001555860042572, 1.5734198093414307, 0.8101712465286255, 0.22843898832798004, 1.0128740072250366, -1.5957655906677246, 0.11038421839475632, 0.7947611212730408, 1.7748924493789673, -0.7827701568603516, 0.17537030577659607, 0.1269301474094391, -0.47658610343933105, 0.8357906937599182, -0.47089624404907227], [-0.9675840139389038, -0.08897716552019119, -1.3022419214248657, 0.9818841814994812, 0.280497282743454, 0.3519893288612366, 0.7191445231437683, -0.2676721215248108, -0.2830095887184143, 0.966722846031189, -0.06099947541952133, -1.7579776048660278, -0.4805745482444763, -0.47622939944267273, -1.643537163734436, 0.13146139681339264, -0.5179879665374756, 1.4941827058792114, -0.49525773525238037, -1.4669374227523804, -1.042464256286621, 0.23168757557868958, -0.9824492931365967, 1.0296279191970825, 1.0713163614273071, -0.3739573657512665, -0.06279473006725311, -1.1898702383041382, -1.0959681272506714, 0.35101985931396484, 0.6691479086875916, 0.7057501077651978, 1.3705978393554688, -1.149857759475708, -0.9406343698501587, 0.6368207335472107, -0.9909575581550598, -0.29095518589019775, -0.567593514919281, 0.8321986794471741, -0.8013739585876465, -1.4123177528381348, 1.7477145195007324, -1.4449844360351562, -0.9072225093841553, 0.5947376489639282, -0.23945847153663635, -0.33351898193359375, 0.38313040137290955, -0.15621671080589294, 0.4974508285522461, -0.787260115146637, -1.198789358139038, 0.6003260016441345, 0.4120471477508545, -0.902747631072998, -0.7672590613365173, -1.7805390357971191, 0.1026936024427414, 0.14478212594985962, 0.14115774631500244, 0.9490054845809937, -0.23225103318691254, 0.3454034924507141, -0.24448339641094208, -1.7594877481460571, 0.9060531258583069, 1.4504152536392212, 0.635977029800415, -1.4806779623031616, 0.24019505083560944, -0.6603947281837463, -1.6381562948226929, 1.1668639183044434, 0.41285765171051025, -0.4736936688423157, 0.6913634538650513, 1.6402747631072998, 1.3783942461013794, 1.1534287929534912, 0.7264190912246704, 0.033378951251506805, -1.2540004253387451, 0.23607033491134644, 0.08559134602546692, -0.9147692322731018, 0.04048249498009682, 0.6822415590286255, 0.8762878179550171, 0.5934129357337952, -0.7410406470298767, -1.1464817523956299, 1.8509591817855835, -2.6738479137420654, -0.003839098149910569, -0.06418384611606598, -0.09561455994844437, 0.750111997127533, -0.6863630414009094, -0.1859898716211319, 1.4096800088882446, -0.4558658301830292, -0.23518186807632446, 1.3122096061706543, 0.03347688913345337, 0.451220840215683, 1.5867571830749512, 1.6716228723526, -0.9797325134277344, 2.6498355865478516, -1.148373007774353, -0.528171718120575, -0.4250430166721344, -0.9257892370223999, 1.5073602199554443, 0.08054304122924805, -0.7288140654563904, 0.3712471127510071, 0.49466732144355774, -0.977692723274231, 0.3912351131439209, 0.10498199611902237, -0.8693530559539795, -0.33206668496131897, 1.1635708808898926, 0.14249955117702484, -1.0005886554718018, 0.48554274439811707], [-0.5310352444648743, 0.9979732632637024, -1.2892332077026367, -0.8686363697052002, 0.8088476657867432, -1.1133651733398438, -0.18633508682250977, 1.7637425661087036, 0.8624670505523682, 0.35359328985214233, -0.2864733636379242, 0.5576390624046326, 0.926260769367218, -2.328258514404297, -1.986417531967163, 0.15113994479179382, 1.0743460655212402, 2.081512689590454, 1.3978166580200195, 0.6915210485458374, -0.567026674747467, 0.30672088265419006, -0.0025529242120683193, -2.2252871990203857, -0.26710236072540283, -0.211704820394516, -0.011092707514762878, 0.24449753761291504, 0.4412457346916199, -0.6755648255348206, 0.5144569873809814, 1.1328964233398438, -1.1943490505218506, -0.6589125990867615, 0.16434946656227112, 0.0990208238363266, 1.7876551151275635, 1.3358306884765625, 0.3220536410808563, -0.804440438747406, -0.2394615113735199, 1.2269902229309082, 0.6855579614639282, -0.3667376935482025, 0.269713819026947, -1.037360429763794, -2.4368114471435547, 1.1337668895721436, -0.9468812346458435, 0.3409383296966553, -1.685095191001892, 1.5111960172653198, 0.8347179889678955, -0.2802542448043823, -1.0591094493865967, 1.8711094856262207, 0.22909189760684967, -0.7099201083183289, -0.2975855767726898, -0.34320905804634094, 1.2966183423995972, 1.7382476329803467, -0.6136311888694763, 0.48982471227645874, 1.5630606412887573, -0.6603551506996155, 1.216564655303955, -0.40449368953704834, -0.4068441092967987, -0.3028734028339386, 1.5968092679977417, -1.2051793336868286, -0.8018410801887512, 1.2934997081756592, 0.3406120240688324, -1.2828806638717651, 0.4322868883609772, -1.116963267326355, -0.908605694770813, 0.897659957408905, 1.9784597158432007, 0.6502656936645508, 0.7288051247596741, 0.5759011507034302, -1.7311391830444336, 1.8363455533981323, -2.2424814701080322, 0.08257964253425598, -0.09411823004484177, -0.22054551541805267, 1.1541293859481812, 1.2990236282348633, 0.1292167454957962, 0.35310497879981995, -0.2745521664619446, 2.5269320011138916, -0.2585340738296509, -0.0602823942899704, -0.9122980833053589, 0.5229213237762451, -0.3019501566886902, -1.174950122833252, 0.17163726687431335, -0.5203589200973511, 0.584011971950531, -1.7569785118103027, 0.19267253577709198, -2.4441707134246826, 0.08659542351961136, 2.013617992401123, 0.019568998366594315, 0.39021217823028564, -2.251171588897705, 0.2533015012741089, -1.032015085220337, 0.48159360885620117, -0.5425240397453308, -1.0856103897094727, 0.7959150075912476, -0.3520989418029785, -0.8968695402145386, 0.478215754032135, 2.35646653175354, -0.816893994808197, 0.7553452849388123, 1.0480128526687622, 1.1913602352142334, -0.9571671485900879], [-0.599689781665802, 1.3798047304153442, 1.0431153774261475, -0.4431646466255188, -0.9655522704124451, 0.7846596240997314, -0.016204791143536568, 0.6596634387969971, 0.07602901011705399, -0.21313448250293732, -0.955129861831665, -0.6055287718772888, -1.474678874015808, 1.5125973224639893, -2.3835909366607666, 1.1106374263763428, -2.26185941696167, -0.7459801435470581, 0.09260619431734085, 1.0400160551071167, -2.025482177734375, 1.2738468647003174, 1.1104636192321777, 1.1620362997055054, 0.02363983355462551, -0.8001760244369507, -0.5064607858657837, -1.1569957733154297, 0.8233392834663391, 1.5341644287109375, -0.09057164937257767, 0.6058758497238159, -1.3932992219924927, -0.2843739688396454, 1.252837896347046, 0.14542636275291443, 0.4560084342956543, 1.0306400060653687, -0.0830092579126358, 0.7902342081069946, -0.05241258069872856, 0.7338742017745972, -0.004896307364106178, -1.657095193862915, 1.0159590244293213, -0.01484318170696497, 1.6259336471557617, -0.21150271594524384, 0.16772232949733734, -1.6684373617172241, -1.2178477048873901, -0.2201671153306961, -0.5014569163322449, 0.04378482326865196, 0.7523376941680908, -1.8757941722869873, -0.2757699191570282, 1.4032137393951416, -1.2667293548583984, -0.1609354019165039, -1.5133204460144043, 1.0679709911346436, 0.7909952998161316, -2.33058762550354, 0.7977433204650879, -0.16051623225212097, -0.7026624083518982, 0.35778096318244934, -1.1850601434707642, 0.09739809483289719, -0.35727402567863464, 0.0675051361322403, -0.4352230727672577, -1.142111897468567, -0.7694932222366333, 0.2650740444660187, -0.47904515266418457, 1.6190261840820312, 0.6003794074058533, 0.23002907633781433, 0.2755168080329895, 2.1277358531951904, 1.6318928003311157, -1.942173957824707, -0.6334441304206848, -0.4450646638870239, -1.5912244319915771, 2.2845122814178467, 0.8394858241081238, -0.2730543613433838, 0.12649081647396088, -0.023980669677257538, 1.4035062789916992, -0.2716987729072571, 0.6552269458770752, 0.08895666897296906, 0.932697594165802, 0.9078844785690308, -0.4888584017753601, 1.019994854927063, 0.7611916661262512, -0.978410005569458, 1.8972376585006714, -0.9471824765205383, 0.20516961812973022, 0.22375847399234772, 0.8392877578735352, -0.35484206676483154, -0.1592586785554886, -0.01976538635790348, 3.172086477279663, -0.594182014465332, -0.016767453402280807, -0.3529300391674042, 1.3014917373657227, 0.9274666905403137, -0.8434690237045288, -0.03210481256246567, 0.9443413019180298, 1.9276870489120483, -0.2640335261821747, -0.7431035041809082, 1.7767258882522583, 0.11132224649190903, 1.2576593160629272, 1.3889018297195435, 0.8313292264938354, -0.893389880657196], [-0.20753034949302673, 1.3389263153076172, -1.08504056930542, 0.7290350794792175, -1.5428524017333984, -0.5028529167175293, 1.5367766618728638, 0.6493490934371948, 1.3585686683654785, -0.5058830976486206, 0.540822446346283, -2.3810184001922607, -0.4233384132385254, 0.123955637216568, -1.7497174739837646, -0.9304232001304626, -0.6945977210998535, 0.07248622179031372, 0.8707059025764465, 0.10437610000371933, -0.450675904750824, -0.415985643863678, 0.4461859464645386, -0.28912994265556335, -1.0236332416534424, 1.6268011331558228, 0.07626799494028091, 0.5304749011993408, -1.66708505153656, -0.45940661430358887, -0.5907819867134094, 1.5400123596191406, 0.5591942667961121, -1.881746768951416, -1.0361731052398682, 0.21553358435630798, 0.724733293056488, 1.1500736474990845, -0.23479779064655304, -0.5072465538978577, 0.016554970294237137, -0.5710819959640503, -0.0603942796587944, 0.6929670572280884, -0.33041250705718994, -1.599756121635437, 0.8990767002105713, -0.7226125001907349, 0.19894547760486603, 1.4639716148376465, -0.40009385347366333, -3.3696701526641846, -1.6640287637710571, 1.7153708934783936, -1.0470317602157593, 0.5511164665222168, 0.0630926713347435, -0.5344865918159485, -1.6011461019515991, 1.0649724006652832, 0.41932278871536255, 1.4525774717330933, -0.09793190658092499, -1.2947689294815063, -0.44082003831863403, 0.03112366795539856, -1.9672273397445679, 0.4063100516796112, -3.296738862991333, 0.0010814964771270752, -0.5552958250045776, -0.02275128848850727, -0.5199277400970459, 0.2694055736064911, -0.5247126221656799, 0.14399954676628113, -1.1500974893569946, 1.0259242057800293, -1.0487112998962402, -0.6284606456756592, 1.029945969581604, 0.40584227442741394, -0.9732052683830261, 0.4114464521408081, -0.6049952507019043, -0.11315543949604034, -0.45874112844467163, -1.0584601163864136, -0.12471634149551392, 1.2139438390731812, 0.7959007024765015, 0.6989324688911438, 0.9174964427947998, -0.4547375440597534, -0.8476712703704834, 0.26136040687561035, 1.1461620330810547, -0.05952645093202591, 1.110465407371521, 0.9980179667472839, 0.7286797165870667, -2.579519748687744, 1.1479490995407104, 0.165994331240654, 1.1493678092956543, 0.5421838760375977, 0.25676921010017395, -0.4250393509864807, -0.1822415590286255, -0.8543760180473328, -1.4385173320770264, -1.2364109754562378, 0.9282928705215454, 0.6122984886169434, 0.001852346700616181, -1.91438889503479, 1.5046272277832031, -0.889115035533905, -0.08083825558423996, -0.36325323581695557, -0.8682292699813843, -0.4260410964488983, 1.3324365615844727, 0.7267586588859558, -0.5523912906646729, 1.4040497541427612, -0.008852191269397736, 0.09188316762447357], [0.005220571532845497, 0.09251866489648819, -0.5436851382255554, 0.05942964181303978, 0.07322545349597931, 0.22586873173713684, 0.06375952064990997, 0.030479177832603455, 1.4594132900238037, -0.7638933658599854, -0.3368484079837799, -0.870731770992279, -1.5392916202545166, 0.6690822839736938, 0.3324800431728363, 0.9009890556335449, -0.030613332986831665, -0.14325366914272308, 0.8321101665496826, 0.48928776383399963, 0.2144613265991211, -0.7943994998931885, -0.3283064067363739, -0.7025181651115417, -0.6191426515579224, -0.30332401394844055, 0.3243163228034973, -0.1770671159029007, -2.001534938812256, -1.0478333234786987, -0.5157373547554016, -0.6030033230781555, 0.899655818939209, -0.2792651057243347, 0.2748441696166992, 0.7189033031463623, 0.946098268032074, 1.5808336734771729, 0.7852492928504944, -0.08532781153917313, -0.4723912179470062, 0.7169989943504333, 1.3525010347366333, 1.4088318347930908, -1.067366361618042, -1.5431976318359375, -0.5313212275505066, 0.600570023059845, -2.483126640319824, 0.5103132724761963, 0.12501350045204163, 0.2534934878349304, 0.24505527317523956, -0.49104833602905273, 0.985643208026886, -2.3700969219207764, -0.5110558867454529, 1.1972864866256714, 1.583574652671814, 0.29181477427482605, 1.0833520889282227, 0.45432114601135254, -1.8214489221572876, -1.2173691987991333, 1.5731556415557861, 0.02091609500348568, -0.4066033959388733, 0.1945527344942093, -0.3336544334888458, 0.4130411148071289, -0.19741667807102203, -0.9384718537330627, -0.6099374294281006, 0.9550833106040955, 0.5488861799240112, 1.0834401845932007, 0.47929731011390686, -1.911191463470459, 0.9153371453285217, -1.9814419746398926, 0.40036115050315857, -0.6827436089515686, -1.3689218759536743, -0.971055269241333, 0.005914811976253986, 0.09192138910293579, -1.2494364976882935, 1.313126802444458, -0.1523282825946808, -0.0781637504696846, -0.16668741405010223, -0.48331907391548157, 0.26779988408088684, 0.43107128143310547, 0.37272313237190247, -0.7626054286956787, -0.5222906470298767, 0.645979106426239, 0.3848637640476227, -0.979439377784729, 1.2797621488571167, 0.019046910107135773, -0.4787876605987549, 0.10796403884887695, 0.6862567663192749, -1.6822106838226318, 1.4601898193359375, -0.5392717719078064, -1.6573940515518188, 0.9307287335395813, 0.4445725083351135, -0.35401833057403564, 0.2762930989265442, 2.447512149810791, -1.3840866088867188, 1.2117788791656494, -0.34822210669517517, 1.5533555746078491, -1.4013087749481201, 0.5410157442092896, -0.3824772536754608, -0.4724447727203369, -0.5016915202140808, 0.37516263127326965, 0.496506005525589, 1.53416907787323, -0.44442811608314514, 0.2259521633386612], [-0.40378081798553467, -1.321879267692566, -0.4839203655719757, -0.27066805958747864, -0.21354179084300995, -0.33454182744026184, 0.5718746185302734, -0.41237184405326843, -0.5870048999786377, -1.031467318534851, 2.579390287399292, -0.3274572789669037, -0.4974539875984192, -0.5486458539962769, 0.5441903471946716, 0.37001803517341614, 0.46260568499565125, 0.11716683208942413, 0.6782493591308594, -0.345333069562912, -1.3357957601547241, 0.32040947675704956, 0.09851615130901337, 0.36796027421951294, -0.1269402652978897, 1.6090457439422607, 0.5759109258651733, 0.16136564314365387, -0.7639476656913757, 0.5348368287086487, 0.34558233618736267, 0.21955399215221405, -0.4653799533843994, -0.5149383544921875, -0.5298518538475037, 1.0533742904663086, -0.5244532823562622, 0.39036744832992554, 0.2640021741390228, 1.0606355667114258, 0.14371609687805176, 0.002233466599136591, 2.3992300033569336, -0.3429439961910248, 0.14947736263275146, 0.539668083190918, 0.5330225825309753, 0.7252955436706543, 0.34465470910072327, -0.6661850214004517, -0.8933312892913818, 0.2183677852153778, -0.0455026850104332, 0.8152270913124084, 0.8112242817878723, 1.555546522140503, 1.9512611627578735, -1.2334403991699219, 0.2321041077375412, -2.401552200317383, 0.41839608550071716, 0.7358672022819519, 0.42953208088874817, -0.10350698977708817, -0.9658184051513672, -0.22585928440093994, -0.5665754675865173, 0.01653568260371685, -0.2286018580198288, -0.5494685769081116, -0.6647138595581055, 1.0083212852478027, -0.4964974820613861, 0.12974314391613007, -2.4701356887817383, 0.6066717505455017, -0.44799256324768066, 0.9406924843788147, 0.1391114741563797, 0.49773672223091125, 1.4468482732772827, -0.7458263039588928, 0.0524798259139061, 2.0068304538726807, -0.3445741534233093, 0.5036101937294006, -0.20026607811450958, -2.47100830078125, 0.7463608384132385, 0.06442989408969879, 0.28606316447257996, -0.9780065417289734, 1.359913945198059, 0.01002667099237442, 0.754296064376831, 0.9449195265769958, -0.6200803518295288, 2.063746929168701, -0.1431407481431961, 0.8364129066467285, -0.3514930307865143, 0.17838314175605774, -0.2086787074804306, -1.4292734861373901, 0.1288624107837677, -2.138017416000366, -0.9743074774742126, 0.013549069873988628, -0.8067446351051331, -0.4420322775840759, 0.7804325819015503, -0.9936470985412598, 0.13480976223945618, 0.8771423697471619, -0.7468798160552979, -0.6909682154655457, 0.5456886291503906, -0.5573576092720032, -1.4045825004577637, -0.12180273979902267, -0.4178282916545868, 1.1483001708984375, 0.2534772753715515, -1.4544548988342285, 0.062190763652324677, -0.593425452709198, 0.7720118165016174, 0.5116046071052551], [0.28888440132141113, -0.5866068601608276, 0.1602792739868164, 0.693302571773529, 0.521016538143158, 1.0647125244140625, -0.6504656076431274, 0.7405588626861572, 0.01369738020002842, 1.0819050073623657, -0.0336768738925457, -0.4189112186431885, -2.0867815017700195, 1.2745085954666138, -0.8144482374191284, 0.3558790683746338, -0.7954681515693665, 1.7780439853668213, 0.4641638994216919, -0.10222043842077255, 0.037120383232831955, -0.3377242386341095, 0.04018058255314827, 0.48605096340179443, -0.33293330669403076, 1.6774240732192993, 0.17378972470760345, -0.3077520728111267, 0.11374492943286896, 1.8696682453155518, 1.7679380178451538, -0.36198917031288147, -1.0221376419067383, -0.07135336101055145, 0.7188333868980408, 0.6210825443267822, 0.3803330659866333, 0.8881651163101196, -1.0150507688522339, -1.0456629991531372, 0.06251394003629684, 0.5236086845397949, 2.0914220809936523, -0.7084536552429199, 0.6899112462997437, 1.9736460447311401, 1.8385083675384521, -0.27834731340408325, -0.0054734437726438046, 1.6159186363220215, 1.4848504066467285, -0.5020974278450012, -1.6229082345962524, 0.07681829482316971, 0.113315150141716, -1.0793603658676147, 0.27090999484062195, 0.0670384019613266, 0.4519684910774231, 0.4793888330459595, 0.297956258058548, -1.8582813739776611, -0.08568426221609116, -1.7188071012496948, -0.43385830521583557, -0.03748931363224983, 1.1627752780914307, 1.3129329681396484, 0.6802713871002197, 0.34065449237823486, 0.5538301467895508, 2.4332096576690674, -0.16715772449970245, -1.5939831733703613, -0.39844921231269836, 0.07086916267871857, 0.07960195094347, 0.6306421756744385, 0.25133293867111206, -0.5911633968353271, -0.9953321814537048, 2.1440060138702393, -0.028811205178499222, -0.7553775310516357, -0.21377331018447876, -0.3678489625453949, -2.8105554580688477, 1.2573403120040894, -0.569837749004364, -0.1721109002828598, 1.2559741735458374, 0.7645497918128967, 1.8279637098312378, -0.4655538499355316, 0.5522035956382751, 0.03608859330415726, -0.7996222972869873, -0.08011220395565033, -0.49444249272346497, -1.951313853263855, 0.855506956577301, -0.5704406499862671, 0.09371039271354675, -0.034577980637550354, 1.4797568321228027, 0.38762980699539185, 0.4516918659210205, 0.29965972900390625, -1.008682131767273, -2.0775060653686523, -1.1310274600982666, 0.3338393568992615, 0.6213861107826233, 3.0439770221710205, -0.45098888874053955, 0.9330103397369385, 0.013357514515519142, 0.8366070985794067, -0.9488815665245056, -0.20375825464725494, 1.4076241254806519, -1.1985931396484375, -0.5579403638839722, -0.18701018393039703, 0.1768784373998642, 1.117898941040039, 0.7202273607254028, 0.9683425426483154], [-1.1936635971069336, -0.833726167678833, 0.5126834511756897, -0.23162759840488434, -0.9128232002258301, 0.0517575778067112, 0.3481510877609253, 0.31533610820770264, 0.2949220836162567, -1.6735808849334717, -0.12292385846376419, -1.4428118467330933, -0.5878299474716187, -0.5504500865936279, -0.5732855796813965, 0.04713593050837517, 0.8117555975914001, -0.7947601079940796, 0.9431254863739014, -2.5195095539093018, 0.18633154034614563, -0.44659411907196045, -0.619827926158905, -1.356999158859253, 1.236632227897644, 0.6224710941314697, 1.6809674501419067, 1.5513851642608643, 1.0727206468582153, 0.6850157380104065, -0.551917552947998, -0.23521311581134796, -0.21891792118549347, -0.12383167445659637, 1.2862296104431152, -1.5529425144195557, -0.10629135370254517, 0.009892215020954609, 0.2882971465587616, 0.5875000953674316, -1.1714441776275635, 1.4367656707763672, 0.7071740031242371, -0.5986575484275818, 1.522405743598938, 0.23183909058570862, -1.759665608406067, -0.15331122279167175, -0.022076869383454323, -0.7907960414886475, -1.6420835256576538, 0.07085511088371277, 0.40499112010002136, -1.1655868291854858, 1.8386787176132202, -1.2918776273727417, 0.36791279911994934, 2.7386186122894287, 2.26711368560791, 0.4448696970939636, 0.1700972616672516, 1.105394721031189, 1.9757732152938843, 0.1936054229736328, -0.35521402955055237, -0.666450023651123, -0.41972070932388306, -2.3592214584350586, -0.9314976930618286, 0.5431875586509705, -1.8222142457962036, 2.0847866535186768, -0.0432993620634079, 1.1229579448699951, 0.08427304029464722, 1.4630184173583984, 1.2119014263153076, -0.5940777063369751, -0.020009750500321388, 0.7860419750213623, -0.9556103348731995, -1.3153131008148193, -0.01357786264270544, 0.8676455616950989, 1.3778706789016724, -0.07441964745521545, -0.28437137603759766, -0.9929860830307007, -0.4559212625026703, -0.9598575830459595, 1.2694119215011597, -0.4027559161186218, -0.22357456386089325, -0.0615864098072052, 1.1972452402114868, -0.36259135603904724, 0.48203712701797485, 0.8367885947227478, 1.319070816040039, -0.9040067195892334, -0.1063695177435875, 1.3853123188018799, 1.740691900253296, 0.3841612935066223, -1.7090425491333008, 0.6775802373886108, -0.04339433088898659, -1.1474226713180542, 0.6294137239456177, -0.7472178936004639, 0.02104754000902176, 0.06424914300441742, -0.5357578992843628, -0.5960375666618347, -1.831666350364685, 0.8074151873588562, 0.9836308360099792, -1.9040626287460327, 1.0731513500213623, 0.773410439491272, -0.04375180974602699, -1.5848631858825684, 0.2554419934749603, 0.8087337017059326, 0.690667450428009, -0.704988956451416, 0.4730563163757324, 0.4170759320259094], [1.2875531911849976, -1.168192982673645, -0.12281013280153275, 0.20120352506637573, 0.6767215728759766, -0.9880077838897705, -1.9047917127609253, 0.3951982259750366, -0.05864809826016426, -0.0026411779690533876, -1.0884935855865479, -0.3413265645503998, -0.6273413300514221, -0.48105770349502563, 0.15416277945041656, 0.5855080485343933, -1.4308137893676758, 1.4856324195861816, 0.11423175781965256, 0.07402516901493073, -0.7805881500244141, -0.6158117651939392, 0.5488871335983276, -0.6112062335014343, 2.4272727966308594, -0.7647629380226135, 1.750801682472229, -0.46835142374038696, 1.3235467672348022, 0.5979113578796387, 0.7624006867408752, 0.3831263482570648, 0.7157405018806458, 0.41520822048187256, 1.2048581838607788, -0.11310611665248871, -0.5118644833564758, 0.21501965820789337, 0.38847002387046814, 1.3803575038909912, -0.23374925553798676, 0.20087288320064545, -0.4927847683429718, -1.4892463684082031, -0.7611749172210693, 0.8899253010749817, -0.5551276206970215, -2.0298879146575928, 0.3847900927066803, 0.261355459690094, 3.628056287765503, 0.3093078136444092, 0.08859843015670776, -1.6394188404083252, 0.32490870356559753, 1.3012099266052246, 1.975040316581726, -1.1788561344146729, -1.1361697912216187, 0.5002658367156982, 0.1497815102338791, 0.4604388177394867, -0.501716136932373, 1.0070879459381104, -1.1685731410980225, -0.1982261687517166, 0.49219968914985657, -1.5551599264144897, 0.9367474317550659, 0.5967350602149963, -0.09872148185968399, 1.4417330026626587, 0.3329324722290039, 0.23193170130252838, 1.3250869512557983, -0.7738732695579529, 0.011970223858952522, 0.5066462159156799, 0.737525224685669, -1.9094130992889404, -0.823104977607727, -2.1160411834716797, 1.8885881900787354, 0.47320133447647095, -0.6771088242530823, -0.8326218128204346, -0.057455938309431076, 0.6051860451698303, 0.02236137166619301, 0.3016325831413269, 1.7828385829925537, 0.634060263633728, -1.9089335203170776, 0.7435222864151001, 2.1557443141937256, -0.0822470560669899, -2.620546579360962, 0.9743567109107971, 0.33018508553504944, 0.9725599884986877, 1.4513797760009766, -2.029459238052368, 1.3212130069732666, 1.0491857528686523, -0.8616384267807007, 1.0944205522537231, -2.03867506980896, 2.7913119792938232, -1.217539668083191, -0.8857897520065308, -0.956088125705719, -0.12858133018016815, 0.38283464312553406, -0.9676647186279297, -0.6708357334136963, -1.9476803541183472, -0.3789007365703583, 0.46867185831069946, 0.9396589994430542, 1.1047933101654053, 1.3135377168655396, -0.5915839672088623, -1.2040473222732544, -0.017088057473301888, -2.0681443214416504, -1.1636505126953125, 0.43948373198509216, 1.4888689517974854], [-0.3363973796367645, -0.26977160573005676, 1.3177566528320312, -1.5800998210906982, -0.37008675932884216, 0.7442012429237366, -1.0299413204193115, 0.050946757197380066, 1.233012080192566, 0.35576701164245605, -0.626954197883606, 0.7965498566627502, -0.3992697298526764, -0.9338247776031494, -0.9054528474807739, -0.8746749758720398, 0.7258586287498474, 0.05774332210421562, -0.21722900867462158, 0.9315308928489685, 0.3135005831718445, 0.10087458789348602, 0.5619313716888428, -0.2821836769580841, 0.19161255657672882, -1.1563810110092163, -1.8828096389770508, -1.5259101390838623, -1.5839781761169434, 0.4163469970226288, -0.30216193199157715, 1.4194263219833374, 0.4717828035354614, 1.644407033920288, -0.8909318447113037, 0.44969525933265686, 0.9978461265563965, -1.237444281578064, -2.4371554851531982, 0.12425092607736588, 2.1562302112579346, 2.4792566299438477, -0.5942758321762085, -0.4114724397659302, 0.8287816047668457, -1.5510821342468262, 0.6247301697731018, -0.8070228695869446, -0.8610561490058899, -0.3247166574001312, -0.24841105937957764, -0.9532990455627441, -0.5872573256492615, -0.994132399559021, 1.085358738899231, -2.0512917041778564, 0.15460239350795746, -1.1754943132400513, -1.5895521640777588, -0.6169967651367188, 1.3605018854141235, 0.2626071572303772, 0.1613936722278595, 0.1408078372478485, -0.13656528294086456, 0.7179415225982666, 0.9585882425308228, 0.08610918372869492, 0.6934661865234375, -0.5810402631759644, 0.9484363198280334, 1.225196361541748, 0.6604206562042236, 1.3226113319396973, 1.1295349597930908, 0.7680504322052002, 0.18587522208690643, -0.032747533172369, 1.0432405471801758, 0.2655785083770752, 0.719266951084137, -0.23943829536437988, 0.08771464228630066, 1.2167471647262573, 0.3024413287639618, 0.193434476852417, -0.1314043551683426, -1.9748862981796265, 1.0529991388320923, 0.053702179342508316, 0.20985101163387299, -0.1273883730173111, 0.4254468083381653, -1.3974577188491821, -0.07225897163152695, -0.06523142009973526, -0.5409975051879883, -0.7646211385726929, -0.1797686070203781, -1.118353247642517, -0.15040193498134613, 2.8175792694091797, 0.62641841173172, -0.2970343232154846, -0.5948350429534912, -0.508906364440918, -2.356708526611328, -0.47260770201683044, -0.463515967130661, -1.3129973411560059, 0.7595486640930176, -0.4286718964576721, -1.2148617506027222, -1.3490960597991943, 0.6155203580856323, -0.8637431859970093, 0.4081306457519531, 0.9335798025131226, 0.7447091341018677, -0.5183354020118713, -0.10761440545320511, 0.6749663949012756, 1.2863590717315674, -0.4680078625679016, -0.08828659355640411, 0.2946632206439972, -0.9615539908409119, -1.219478726387024], [1.5917856693267822, 0.7284156680107117, -0.367793470621109, 0.7416527271270752, 0.2565848231315613, 0.8174880146980286, 0.1366463005542755, 0.30048057436943054, -1.0266451835632324, -3.1607768535614014, -1.129581093788147, -0.06788461655378342, -0.19252772629261017, -0.5317206382751465, -0.26343002915382385, 0.1282954216003418, -0.16582992672920227, -1.1131415367126465, 1.41361665725708, -0.5824548006057739, 0.6529858112335205, -0.336608350276947, -1.0920076370239258, -0.3889177739620209, -1.3359507322311401, -0.7205477356910706, -1.846193790435791, 2.2786734104156494, 0.07620322704315186, 0.28335291147232056, -0.7627835273742676, -1.5475248098373413, -0.13952194154262543, -0.458202064037323, 1.261029839515686, 1.7764006853103638, 0.07059625536203384, -0.5367347598075867, -0.6415704488754272, -0.058359790593385696, -1.1330533027648926, 1.2173763513565063, 0.0037436154671013355, 0.597237229347229, 0.11829816550016403, -1.1801363229751587, 0.6220168471336365, 0.39458200335502625, 1.3597279787063599, 0.30935755372047424, 0.5353837013244629, 0.05058586969971657, -0.1577034443616867, 0.13983453810214996, 0.20205968618392944, 1.207863688468933, 2.584200143814087, -0.7676609754562378, -0.7827709913253784, 2.0455198287963867, -0.22293724119663239, 0.5949972867965698, -0.4387100338935852, -0.1726098358631134, -0.0735311508178711, 0.05523717775940895, 1.8071790933609009, 0.6152368783950806, -0.5095621347427368, 0.5773578882217407, 0.20673038065433502, -0.6471989154815674, 0.12283367663621902, 0.18213146924972534, 0.3361331522464752, 0.33502012491226196, -0.17632165551185608, -0.9909677505493164, 0.075745590031147, 0.8275831937789917, 0.2876111567020416, -2.141190767288208, 0.5739312171936035, -0.8958147764205933, 0.6312036514282227, -0.2669818103313446, -0.37201234698295593, -1.7868797779083252, 0.9944814443588257, -0.7960442900657654, -0.20505556464195251, 0.37802842259407043, -0.5561161637306213, -1.7825968265533447, -0.2921326756477356, 0.5790577530860901, 0.983555257320404, -0.939784586429596, 1.5461660623550415, -0.13970215618610382, -0.7805594205856323, 0.9241055250167847, -0.7166461944580078, -1.3614784479141235, -1.594089150428772, 1.0275053977966309, -2.4914424419403076, -2.042206048965454, 0.9472874999046326, 1.5892177820205688, 0.5436229705810547, 0.046265218406915665, 0.751183032989502, 0.25759461522102356, -0.9298903942108154, -0.783897340297699, -0.27534720301628113, 2.2242510318756104, 1.9093660116195679, -0.21457800269126892, -1.1014443635940552, 0.49189838767051697, -1.1925235986709595, 0.3544575870037079, -0.159718856215477, 0.009680412709712982, -0.12871746718883514, -1.187351942062378], [-1.3328622579574585, -1.2455179691314697, 0.6693905591964722, 0.07957448810338974, -0.4475753605365753, -1.226396918296814, 1.3485968112945557, -0.1840602606534958, 0.8857626914978027, -0.5693506598472595, 1.2052717208862305, -1.0420018434524536, -1.4984668493270874, 0.5233270525932312, 0.053494155406951904, 0.10019967705011368, 0.23374336957931519, -0.1946631669998169, -0.39921826124191284, 0.03741887956857681, -0.6685444116592407, -2.7407233715057373, 0.6627411842346191, -0.37528327107429504, -0.5697662234306335, 0.4079528748989105, 1.423917293548584, 0.0372125543653965, 0.26906076073646545, -0.003397422842681408, -0.2579014301300049, -1.3749420642852783, 0.5970349907875061, 0.4336403012275696, -0.4023255705833435, 0.3684830963611603, 0.4140196144580841, 0.7723495960235596, -0.6962589025497437, -0.7403298020362854, 1.2665668725967407, -0.3404725193977356, 0.8278546333312988, -1.1252448558807373, 0.32981735467910767, -0.8294464945793152, 0.9126453399658203, 2.221510648727417, 0.47038233280181885, 1.815760850906372, -0.7545296549797058, 0.7574813365936279, -0.9284238219261169, -0.22860993444919586, 0.4506261646747589, -0.7126430869102478, -0.9631595015525818, -1.8807923793792725, 0.12035314738750458, -1.0913654565811157, -1.225767970085144, 0.36671602725982666, -0.5712437033653259, 0.034992508590221405, 0.42339298129081726, -1.5699408054351807, -1.249396800994873, -0.6682298183441162, -0.6109892725944519, -0.1199907660484314, -0.8524981737136841, -1.4390147924423218, 0.2321161925792694, -0.36019688844680786, 0.7027862668037415, -0.003476398531347513, -0.6572201251983643, 0.6342712640762329, 0.32068660855293274, 1.523894190788269, 0.11135829240083694, -1.0344878435134888, 0.9169995188713074, 0.09167113900184631, -0.5630279183387756, 2.5764572620391846, 1.5974407196044922, -0.05807389318943024, -1.541632056236267, 1.1668418645858765, 0.40412279963493347, -0.629951536655426, 0.8920204639434814, -0.3178703784942627, -0.33884096145629883, -0.24307945370674133, -0.2107595056295395, 0.10519540309906006, 1.0620169639587402, 0.4757576584815979, 0.8713993430137634, 0.028704650700092316, 0.041377995163202286, 0.43280014395713806, -1.636413335800171, -0.022786181420087814, 0.763875424861908, -0.639141857624054, 1.4524635076522827, -1.0926765203475952, 0.7878145575523376, -0.07311831414699554, -1.074854850769043, 0.09429693222045898, -0.3974568545818329, -0.11375638842582703, 0.2002890408039093, -1.3679478168487549, 2.4432311058044434, 1.2811877727508545, -0.8899930119514465, 0.7325131297111511, 1.7625590562820435, -0.878197431564331, 1.0326831340789795, 0.9175485372543335, 0.199118971824646, -0.08842679113149643], [-2.6822292804718018, -1.283219814300537, 0.5880749821662903, 1.0586618185043335, -1.2739524841308594, -0.1310848891735077, -0.9686300754547119, -0.36353859305381775, -0.3524061143398285, -1.7549529075622559, -0.01633806899189949, 0.3387318551540375, 0.46264177560806274, 1.049930453300476, 0.46634912490844727, 0.3792836368083954, 0.40515825152397156, 0.05337929353117943, 0.5740495324134827, 1.0710257291793823, -0.35559937357902527, -0.46885624527931213, -0.8504054546356201, 0.8747239112854004, 0.035785235464572906, 0.2297043800354004, 0.21878570318222046, 0.43262049555778503, -1.758481740951538, 0.5618868470191956, -0.8024112582206726, -0.5361002087593079, 0.5160380005836487, 0.3995437026023865, 1.7563560009002686, -1.042222499847412, -0.5008075833320618, 0.5150162577629089, -0.5317471623420715, 0.553753674030304, -0.32973477244377136, -0.5177100896835327, -1.9194414615631104, 0.860209047794342, 0.9658453464508057, 1.7114317417144775, 0.5830729007720947, -1.2117723226547241, 0.17526015639305115, 0.6778084635734558, -0.3438817858695984, 0.22533218562602997, 1.9702767133712769, 0.5074809789657593, -0.37618929147720337, -0.35581186413764954, 0.07921086996793747, -0.8482910394668579, 0.9689446687698364, 0.5762333273887634, 0.21960264444351196, -0.7699545621871948, -0.03629766404628754, 1.0028834342956543, -1.7210620641708374, -0.3451308012008667, 0.37892961502075195, 0.5406364798545837, -1.3514328002929688, -0.19527018070220947, 0.7319114208221436, -1.1134682893753052, 0.8206345438957214, -0.7783151865005493, -0.06456564366817474, -0.996026873588562, 0.6306498646736145, 2.4295506477355957, -0.1806716024875641, 0.035463061183691025, -0.18483878672122955, 0.6609532237052917, -0.7715031504631042, -0.4388806223869324, 1.8797857761383057, 0.5177565813064575, -0.7829561233520508, -0.9449455738067627, -1.0979657173156738, -0.38303542137145996, -0.21206626296043396, 0.025886405259370804, 1.065293312072754, -0.9708934426307678, -0.9250289797782898, 0.6082362532615662, -0.16002660989761353, 0.72447270154953, -0.08731041848659515, -0.6808103322982788, 0.08062275499105453, -1.00186026096344, -1.0192344188690186, -1.0767626762390137, 0.1081453487277031, -1.5444563627243042, -1.5893384218215942, -0.1260843724012375, -0.5246180891990662, 2.40354585647583, -1.900143027305603, 0.8965290188789368, 0.08607923239469528, 0.441391259431839, -1.9267468452453613, -0.016268812119960785, 0.10622826218605042, -2.4716014862060547, 0.2525452673435211, 1.445751428604126, -0.7631171941757202, 0.8140339851379395, 1.8312969207763672, 0.42757648229599, -0.9037501811981201, -0.7785807251930237, 1.1204419136047363, -0.8165149688720703], [0.8636767268180847, -0.7885255217552185, 0.21703435480594635, 1.2691751718521118, -0.20160016417503357, -0.9217184782028198, -0.5797685980796814, 0.09455705434083939, 1.7155160903930664, -1.6741375923156738, -1.3036613464355469, -0.08671282976865768, 0.32821154594421387, 0.9968942403793335, -0.07977791130542755, -1.475748896598816, 2.1378061771392822, -0.47835907340049744, -1.8893922567367554, -0.9873400330543518, -2.236645460128784, -0.6228145360946655, -0.35295361280441284, -1.7602767944335938, -1.7943693399429321, 0.554712176322937, -0.4936012327671051, -0.4398025572299957, 1.4092292785644531, -0.6520885825157166, 0.5842642784118652, 1.3582584857940674, -0.6901649236679077, -0.8655229806900024, -0.2171509861946106, -1.0523453950881958, 0.10848021507263184, 1.0950766801834106, -0.26721957325935364, -1.6026325225830078, -2.724442481994629, -1.0025466680526733, -0.16062100231647491, 0.6766291260719299, -1.1173970699310303, -0.2211543470621109, -1.0770814418792725, -0.5555038452148438, -0.45909541845321655, -0.267214298248291, 0.08101115375757217, -0.44283074140548706, 0.6406290531158447, 0.5281645059585571, 0.2926884591579437, -0.8423601984977722, 1.2671211957931519, -0.17153462767601013, 0.06463447213172913, 1.3834478855133057, -0.009755118750035763, -0.7994958162307739, 1.2270878553390503, 0.14617487788200378, 0.18753761053085327, 0.6664932370185852, -0.28041908144950867, 1.3053690195083618, -0.9134758114814758, 1.9215706586837769, -1.4509080648422241, -1.7159818410873413, 0.5346906781196594, 1.2936478853225708, -1.0955408811569214, 1.679485559463501, 2.260903835296631, 0.7154737710952759, -0.5154518485069275, -0.5724881291389465, 0.7138680219650269, 0.7221677303314209, 1.026999592781067, -1.5520780086517334, -0.028193065896630287, -2.1705965995788574, -0.5243986248970032, 1.6465719938278198, -0.15058943629264832, 0.535528302192688, -0.2165827453136444, -1.745382308959961, -1.148349404335022, -1.2519099712371826, -0.09046326577663422, 0.4586709141731262, -0.073271244764328, 0.085360586643219, 0.28088459372520447, -0.39861786365509033, -0.6436163187026978, -1.5355784893035889, 0.04068649560213089, 0.6734519600868225, -0.36814191937446594, -0.623533308506012, 0.9987263679504395, -0.5231866240501404, 0.6835430860519409, 0.9195486307144165, -0.06030784174799919, -1.8295649290084839, 0.6537225842475891, -1.6979341506958008, -1.8133388757705688, 0.15401394665241241, 0.17931964993476868, -0.45600613951683044, -0.5809674859046936, 1.349733591079712, -0.4440072178840637, -0.182834655046463, -0.03735658526420593, -0.9557289481163025, 1.8126766681671143, -0.483023464679718, -1.090812087059021, -0.24723325669765472], [1.1628819704055786, -2.884859800338745, 1.1767946481704712, -1.811696171760559, 0.7686130404472351, -1.053409218788147, -0.712454080581665, -0.6836146116256714, 0.07806327193975449, -0.3136974275112152, 0.4833444058895111, -0.8057482242584229, -0.9396834373474121, -0.9590671062469482, -1.4345132112503052, 1.962743878364563, -1.4269731044769287, -0.38042521476745605, -0.2139734923839569, -0.5946629047393799, -1.2476770877838135, 0.6921463012695312, 0.6407129764556885, 0.08697815984487534, 0.14085394144058228, 2.5160396099090576, -0.8925973176956177, -0.8768798112869263, -1.8325157165527344, 1.7314984798431396, -0.17630410194396973, 1.595010757446289, 0.49686339497566223, -0.747276246547699, 0.9298828840255737, 0.5753633379936218, 0.07927396893501282, -0.0669383704662323, 1.3967387676239014, 1.7617218494415283, 0.6031410694122314, 0.2460787445306778, 0.3393939733505249, -0.27040407061576843, 1.500708818435669, 1.0366289615631104, -0.18214210867881775, -0.1584901064634323, 0.13530834019184113, 0.23770248889923096, 0.2475871443748474, 0.8839501738548279, 1.344763159751892, 0.810399055480957, 0.3636815547943115, 0.7183451056480408, 0.6245711445808411, -0.2209070324897766, 0.387658029794693, -0.9475402235984802, -0.033290423452854156, 0.42325374484062195, 0.023724982514977455, -0.7131056785583496, 0.9798415303230286, -0.3007260262966156, 0.19150753319263458, 0.1360742300748825, 1.7799334526062012, 0.20176967978477478, 0.8263869881629944, -0.07800287008285522, 0.8790539503097534, -0.8149766325950623, -1.846347451210022, 0.6475018858909607, 1.1273326873779297, -0.24920061230659485, 0.2638370096683502, -0.2302238494157791, 0.3547617197036743, -0.8780379295349121, -0.2811872959136963, 1.4763422012329102, -1.2644551992416382, 0.6770315766334534, -1.0975978374481201, 0.3183385133743286, -1.3845099210739136, -0.6112096309661865, -0.6438496112823486, -2.4846384525299072, 1.363906979560852, 1.1704436540603638, -2.158076286315918, -0.4513823390007019, 0.8788386583328247, -0.5690927505493164, -0.3463752269744873, -1.7794109582901, 0.0752711221575737, 0.607671856880188, -1.7371516227722168, -0.30347487330436707, 0.31614020466804504, 0.7089557647705078, 0.3721287250518799, -1.161034107208252, 0.7221329212188721, 1.4473786354064941, -0.22835582494735718, -2.1199028491973877, -2.289767026901245, 2.4856698513031006, -2.2516322135925293, 1.3971327543258667, -0.5285748839378357, -0.4325672388076782, -0.38116025924682617, -1.188938021659851, 1.2719275951385498, 0.22310569882392883, -1.2438791990280151, 0.15701881051063538, -0.05048763006925583, -0.9612306952476501, -0.21662671864032745, -1.6555876731872559], [1.0838459730148315, 0.8534111976623535, 1.555675745010376, -0.5625017285346985, -0.6598206758499146, -0.5291804671287537, 0.7032952308654785, -0.3624475598335266, 0.014856467954814434, -0.6367495059967041, 1.3103508949279785, -0.1782844364643097, -0.6188283562660217, 0.28496357798576355, -0.36763623356819153, -0.20305661857128143, 1.3823577165603638, -1.7348304986953735, -0.032979294657707214, 0.016786394640803337, 1.3373280763626099, -0.5757356882095337, 0.7983928918838501, 0.6889593005180359, 0.24770238995552063, 0.24974411725997925, -0.5630154609680176, -0.6213963627815247, 0.6635025143623352, 0.9912076592445374, -0.362537145614624, 0.6492491960525513, 0.40415847301483154, 0.5252935290336609, 0.09751324355602264, -0.7034393548965454, -1.5084952116012573, 0.9680293202400208, -1.9340331554412842, 0.941637396812439, -0.8748734593391418, -0.17733371257781982, 0.025644641369581223, 0.2913360595703125, -0.665499746799469, 0.5664812922477722, 0.738105058670044, 0.8218844532966614, 1.3961617946624756, -1.9432706832885742, -0.7548326849937439, 0.9601895809173584, 0.024167895317077637, 1.681728720664978, 2.0019545555114746, 0.38022348284721375, -1.514849305152893, -0.816076397895813, 0.9495342969894409, -0.4994799494743347, -1.0250988006591797, 0.6379600167274475, 0.8297111392021179, 0.4647338390350342, 0.6232131123542786, -0.46635690331459045, 1.9654567241668701, -0.18294809758663177, -2.357433795928955, -0.8849310278892517, 0.2651071548461914, -1.7087249755859375, -0.7086053490638733, 0.429943323135376, -1.0171595811843872, 0.5680363178253174, 1.150056004524231, -0.7177767753601074, -0.8907848000526428, -0.5360509157180786, -0.11781682074069977, -0.6517043709754944, 0.6071876287460327, 1.2042224407196045, -0.56056147813797, 1.3975646495819092, -1.1460270881652832, -1.0445832014083862, -0.5451844930648804, 0.5946352481842041, -1.7588483095169067, -0.4152529537677765, -1.1460788249969482, -0.3479855954647064, 0.5195619463920593, -0.860106348991394, 0.5796586275100708, 0.9483630061149597, 0.5801347494125366, 0.0899987742304802, -0.46781063079833984, -1.311052680015564, 1.1853817701339722, -0.9957860112190247, 0.46887949109077454, -0.047160662710666656, 0.708524227142334, 0.3772416412830353, -0.3899801969528198, -0.35149645805358887, 0.4644987881183624, 2.612544059753418, 0.1309128701686859, -0.6049224138259888, 0.5801020860671997, -1.0264501571655273, 1.6341073513031006, -1.0705527067184448, 0.002197175519540906, -1.5510796308517456, -0.24199287593364716, 0.9758552312850952, 1.0667189359664917, -0.2586480677127838, 2.439218759536743, -2.4052541255950928, -0.8201248645782471, -0.565889835357666], [0.16284134984016418, 1.269098162651062, 0.38108184933662415, -0.08704628050327301, -0.3433389961719513, -0.8809096813201904, -0.7175902724266052, -1.1011964082717896, 1.7424237728118896, -1.6367453336715698, 1.6130050420761108, -0.795543372631073, 0.3230423629283905, 0.6352686882019043, 2.6040375232696533, 0.12254011631011963, 0.08067192882299423, -0.5304260849952698, -1.4318478107452393, -0.6419383883476257, 0.8085657358169556, -0.017502903938293457, 0.9536073207855225, 0.15184910595417023, 0.5316066145896912, -2.586042642593384, -1.367991328239441, -1.056514859199524, 0.3833232820034027, 0.8053098320960999, -1.4470405578613281, 1.1051867008209229, -0.1208539754152298, 1.5699976682662964, -0.14294202625751495, -0.6362648010253906, -1.2582099437713623, -0.22465211153030396, -0.20023353397846222, 0.30581608414649963, -0.8348308205604553, 1.2516714334487915, -0.3964468836784363, -0.40020498633384705, 0.5803450345993042, -0.004310987889766693, 1.144355058670044, 2.6654105186462402, -0.16759298741817474, 0.5150274634361267, 1.744295597076416, 0.27878597378730774, -0.978243350982666, 0.7247764468193054, 0.987295389175415, 0.3233948349952698, 0.7600394487380981, 0.43635237216949463, -1.2625393867492676, -0.08349768072366714, 1.5251712799072266, -0.5658084154129028, 0.08036836236715317, -0.49716857075691223, -1.1007157564163208, -0.7441757917404175, -0.8847998380661011, -0.8770734667778015, -1.5221210718154907, -0.11826969683170319, -0.647783100605011, -1.8605737686157227, -0.11612295359373093, -1.802621603012085, -2.175126314163208, 0.034841470420360565, 1.4608173370361328, 1.9537709951400757, -2.061802625656128, -0.4744875133037567, 0.005088237579911947, 1.0673962831497192, 0.7482571601867676, 0.8389962911605835, 1.703251838684082, 1.105377197265625, 0.8837214708328247, -0.6220988631248474, -1.311279535293579, -0.03932754695415497, 0.44298848509788513, 0.33326148986816406, -0.47931450605392456, -0.7392563223838806, 1.8647668361663818, 0.5689531564712524, -1.4813776016235352, -1.0129899978637695, -0.5227867364883423, -1.047481894493103, 0.7358501553535461, -1.1043514013290405, -1.4084606170654297, -0.6290997266769409, -0.48785099387168884, -0.4771713316440582, -1.2253642082214355, -1.004721760749817, -1.5416405200958252, 0.9589874744415283, 0.4596441686153412, 0.7455124258995056, -0.05335557088255882, 0.4593876898288727, -0.8477275371551514, 1.4193464517593384, -0.6114919185638428, -0.4773297905921936, 0.13468009233474731, -0.8851759433746338, -1.5822409391403198, -2.9639437198638916, -0.14402200281620026, 0.51697838306427, 0.5301941633224487, -1.8771135807037354, 1.6755249500274658, -0.05860744044184685], [-0.5847204327583313, -0.4723355174064636, 0.7984519600868225, 0.8930484652519226, 1.161597728729248, 0.8602150082588196, -1.3502591848373413, 1.3449287414550781, -1.5469707250595093, 0.9886939525604248, -0.4237498939037323, 1.3238641023635864, 1.6565347909927368, 1.453568935394287, 1.4401905536651611, -0.4091184735298157, -0.08609944581985474, -0.5214450359344482, -2.8479342460632324, 0.22362133860588074, 0.06321784108877182, 0.2735210359096527, 1.0775256156921387, -0.20105525851249695, -0.6350007653236389, -0.6053737998008728, -1.7125587463378906, 1.3419129848480225, -0.8903622031211853, -0.5052846074104309, 0.6231683492660522, 0.09306970238685608, 0.19585967063903809, -1.6591275930404663, 0.02245546318590641, -0.29001563787460327, -0.590531587600708, 0.829010546207428, 0.35430410504341125, -0.20244364440441132, -0.8454909920692444, -0.49260053038597107, 0.6465104818344116, -0.33777716755867004, -0.6765355467796326, 1.4917829036712646, 1.324765920639038, 0.5392838716506958, 0.6704947352409363, 0.4168548583984375, -0.11648687720298767, 2.544917345046997, 0.8183068037033081, -0.14418506622314453, -1.5787501335144043, -0.5712451338768005, -1.1841450929641724, -0.8480871319770813, -1.5016177892684937, 0.4540044367313385, -0.8289206027984619, -0.5374387502670288, -0.6251757740974426, -1.3939199447631836, -0.8799010515213013, 0.5888693928718567, -0.023564159870147705, -0.1533026546239853, 1.0658106803894043, -0.6090770959854126, 0.8004617691040039, -0.2362890988588333, -0.7491961717605591, 1.2778372764587402, 0.4495754539966583, -0.41418683528900146, -0.5435444116592407, -0.2313576191663742, 0.7080010771751404, 0.6788956522941589, -0.03437753766775131, -0.09653575718402863, -0.44115108251571655, -0.4893222451210022, 1.3208186626434326, 1.7987874746322632, 0.8000690340995789, -0.11700908094644547, -1.481109380722046, -0.015241440385580063, -0.16477182507514954, -2.192368268966675, -0.6219993829727173, 0.16376648843288422, 0.7510743141174316, 0.4181540012359619, -0.11382555216550827, -2.00042986869812, 0.5193500518798828, 1.2576500177383423, 1.6577402353286743, 0.8708385825157166, -0.8686188459396362, 0.5687186121940613, -0.6964444518089294, 1.051090955734253, 0.4212965667247772, -0.7048578262329102, -1.2278562784194946, -1.4474807977676392, 0.378044456243515, 0.4144945442676544, -0.7025991678237915, 0.2514025866985321, -1.503019094467163, -0.5621694922447205, -0.20621372759342194, 0.3302309811115265, -0.11628852039575577, -0.2752911150455475, 0.9942213892936707, 1.75813627243042, -0.3501875400543213, -0.601731538772583, 0.44714120030403137, -0.8306583166122437, -0.5999729037284851, 1.4778097867965698], [-0.16419106721878052, -0.4954470992088318, -0.35349801182746887, -1.5290946960449219, -1.3574124574661255, 0.6586084365844727, -0.7441736459732056, 0.659426748752594, 0.1331118494272232, -0.9160357117652893, 0.9004057049751282, -0.013711711391806602, -0.29928603768348694, 1.4109469652175903, -0.07249501347541809, -0.5189962983131409, -0.16059449315071106, 0.6182286739349365, 1.3111144304275513, 1.0917754173278809, 1.1430917978286743, -0.8236160278320312, 3.054002285003662, 0.30365610122680664, 0.455392986536026, -0.7718530297279358, 2.0308523178100586, 1.8636773824691772, -1.6694433689117432, -0.09046895802021027, -1.6509233713150024, -0.01130887120962143, -0.7416080236434937, 0.6137468218803406, -0.147908553481102, 0.5429269671440125, -1.8985648155212402, -0.013161590322852135, -0.42734041810035706, 0.13221651315689087, 1.1866811513900757, -0.23635628819465637, 0.4918438792228699, 0.29244333505630493, -0.2603217363357544, -0.09766535460948944, 0.2017715722322464, 0.2722053527832031, 1.0300017595291138, 0.20048080384731293, 0.22282128036022186, 0.30946794152259827, 0.5732274055480957, 0.24327950179576874, 0.43603187799453735, -1.7079951763153076, -0.0024466542527079582, 0.5710340738296509, 0.6276984810829163, 0.6625825762748718, -2.3818249702453613, -0.6392351388931274, 0.008421416394412518, 0.3636913001537323, -0.11440177261829376, 0.7655221223831177, -1.376967191696167, 0.7633156776428223, 0.5862299799919128, 0.696473240852356, 2.4639790058135986, 0.2307230532169342, 0.6646438241004944, -0.2694985866546631, 0.20185503363609314, 0.5449462532997131, 1.3492448329925537, -0.549990177154541, -0.5346759557723999, 0.9620906114578247, 0.19361937046051025, 0.15872347354888916, -1.4184588193893433, 1.3129758834838867, -2.9316396713256836, -1.6904451847076416, -0.32519325613975525, 0.2145494818687439, -0.2676584720611572, -0.6235032677650452, -0.7071678042411804, -0.6762033104896545, -0.7709305882453918, 0.5879572629928589, 0.2988879680633545, 3.2708523273468018, 0.3429909348487854, -0.75005704164505, 0.3490079343318939, -0.3351422846317291, -0.5593899488449097, -2.1389122009277344, 0.6163609623908997, 0.40855705738067627, 1.6565815210342407, 0.45864710211753845, -0.7450583577156067, 1.35478937625885, -1.1741386651992798, -0.700772762298584, -0.4114452004432678, -0.11750221997499466, 0.30426084995269775, -1.3923834562301636, -1.4175599813461304, 0.7798365354537964, -0.9542174935340881, 0.3421267867088318, 2.4094462394714355, 0.18552841246128082, -1.444615364074707, 0.7034654021263123, -0.6612462401390076, -0.4392918050289154, -0.06525634974241257, -0.10661446303129196, 0.8905941247940063, 0.38612109422683716], [1.174759864807129, -0.1984720230102539, 0.385995090007782, -0.8302868604660034, -0.6941043138504028, 2.1326818466186523, 1.2812926769256592, -0.15716488659381866, 1.0417661666870117, -0.6827642917633057, -1.184313178062439, 1.3841757774353027, -1.0075595378875732, -1.1377229690551758, 0.48720842599868774, 0.7516700029373169, 1.093976616859436, -1.0742496252059937, 0.6317984461784363, 0.7201977968215942, -1.437820315361023, -0.5447390675544739, 0.08347184956073761, -0.3323792815208435, 1.2349088191986084, 0.2629074454307556, 0.7131466865539551, 1.493390679359436, -0.9306601285934448, 0.4703596532344818, -0.26388585567474365, 1.0463602542877197, -1.3346832990646362, -0.9332764148712158, 0.6409483551979065, 0.34488558769226074, 1.1018198728561401, 1.4693055152893066, 0.3738521635532379, -1.0949499607086182, 0.15706782042980194, -0.12404856830835342, 1.430465579032898, -0.1792353391647339, 1.4474257230758667, 0.3656076490879059, 0.659922182559967, -0.7509600520133972, 1.0631154775619507, 0.15184341371059418, -1.0311006307601929, -0.8767987489700317, 0.2058938443660736, 0.2412225753068924, 3.2590408325195312, -0.8327195048332214, 0.34170782566070557, 1.2927982807159424, 0.30893653631210327, -0.059440359473228455, -0.7294021248817444, 0.647040069103241, 0.47898632287979126, 1.3962088823318481, -0.17044876515865326, 2.3875248432159424, -1.3473341464996338, -0.9940778017044067, -0.3452073931694031, 0.05276072397828102, -0.20934122800827026, 1.6915092468261719, 0.01819867081940174, 0.8297351598739624, -0.556074857711792, 2.126052141189575, -0.48689326643943787, 0.18012377619743347, 0.5916167497634888, 1.1065168380737305, -0.8272897005081177, 0.8507338762283325, -0.5355073809623718, -2.106796979904175, -0.7886239290237427, 0.3016127347946167, -0.05214795842766762, 0.21304093301296234, 0.0900835245847702, -1.583509087562561, -1.2612844705581665, 0.8206032514572144, -1.1053521633148193, -0.9710742235183716, 1.1432067155838013, -0.14502878487110138, -0.2814350426197052, -0.06327197700738907, -0.15616987645626068, 0.4810309112071991, 0.19095838069915771, -1.2500962018966675, 0.38446110486984253, -0.06101750209927559, 0.4161451756954193, -0.7818183898925781, -0.6955012083053589, 0.04417579248547554, -0.02401597611606121, 0.33527228236198425, 1.4900412559509277, 0.4902579188346863, 0.6866147518157959, 0.8599972724914551, -2.021524667739868, 0.0257373359054327, -1.5631173849105835, 1.3566200733184814, -0.4182371497154236, 0.8369268178939819, 0.28893184661865234, 1.0473659038543701, 0.46547064185142517, 0.20814457535743713, -1.1208105087280273, 0.767794668674469, -1.3985697031021118, -0.585385799407959], [0.5119112730026245, 0.1858423799276352, 1.4952304363250732, 0.8845105171203613, 1.196502447128296, -1.1037894487380981, 0.4628582000732422, -0.46977895498275757, -0.7602809071540833, -0.5695205330848694, -1.6978200674057007, -0.020779838785529137, 0.45245352387428284, 1.6213387250900269, -0.38692864775657654, -0.2964158356189728, -0.8187623620033264, 1.3023430109024048, -0.4101093113422394, -1.2204948663711548, -0.7198843955993652, 0.9158459901809692, 0.45398661494255066, -1.3548556566238403, 1.026632308959961, 1.8454399108886719, -0.6410746574401855, 1.0584908723831177, -0.041016556322574615, -0.6031590700149536, 0.10378996282815933, -0.9527600407600403, 0.20981557667255402, -0.23626989126205444, -0.3291676640510559, -0.2007828801870346, -0.6380840539932251, -1.560124158859253, 1.8832541704177856, 0.45240339636802673, 0.16392286121845245, 0.38462895154953003, -0.2700069844722748, -0.021158168092370033, -0.4203889071941376, 1.7086312770843506, -0.35839158296585083, -0.6918678283691406, 2.28501296043396, -2.292022943496704, 2.2002370357513428, -0.4719434380531311, -1.2056820392608643, -1.525365948677063, -0.43403157591819763, 1.076427698135376, -0.849483847618103, -0.12276807427406311, 0.12943711876869202, -0.5848707556724548, -0.46553558111190796, -0.44665831327438354, -0.7270340919494629, -1.416480302810669, 2.160862684249878, -0.36056655645370483, 0.1605435013771057, -0.8887892365455627, -0.5086060166358948, -1.4552644491195679, -1.905051827430725, -1.4760712385177612, -0.8503946661949158, 1.1321501731872559, 0.919573187828064, 0.5511624217033386, -0.8751561045646667, -0.22043006122112274, 1.9542313814163208, -1.0532432794570923, 1.037422776222229, -1.4614195823669434, -0.7049126029014587, -2.6486024856567383, -0.6281515955924988, -0.0023272691760212183, 1.3815349340438843, -3.0308611392974854, -1.3956671953201294, 0.7377316951751709, 0.05867486447095871, 0.1571967601776123, -0.6744539141654968, -0.5090218782424927, -1.1022318601608276, -0.9083011746406555, 0.43486514687538147, 0.715440571308136, -1.0038959980010986, -0.28802183270454407, 1.5611048936843872, 2.330977201461792, -2.5836479663848877, 0.4398203492164612, 0.569857120513916, 2.1778016090393066, 0.7585882544517517, 0.8557271361351013, 0.24795827269554138, -0.5007612705230713, -0.6708087921142578, 0.49653029441833496, 1.1560847759246826, -0.5705828666687012, 0.9150184392929077, 0.46814823150634766, -0.6923663020133972, 0.7890945672988892, -1.305985450744629, 0.7394944429397583, 0.35318711400032043, 0.27722984552383423, -2.894261360168457, 0.8975138068199158, -0.6502333283424377, -0.11400812864303589, -0.3109816014766693, 0.2822537124156952], [0.25917837023735046, 0.6996849179267883, -1.2588521242141724, 1.3160176277160645, 0.6729167103767395, -0.09385993331670761, 0.3888896405696869, 0.817793071269989, -0.25494545698165894, 0.3924514353275299, -1.007385492324829, -0.8120245337486267, -0.46242570877075195, -0.7253072261810303, 0.9716701507568359, 1.3778159618377686, 1.812731385231018, -0.8473528623580933, 1.022578477859497, 0.0693087950348854, 0.28906023502349854, -1.041996717453003, 0.32311320304870605, 0.6337449550628662, -0.16843648254871368, -0.4165940284729004, -0.4677470624446869, 0.08905594050884247, -0.9420294165611267, -1.0530563592910767, -1.9238543510437012, -0.7667838931083679, -1.5902854204177856, -0.171530619263649, 1.2740483283996582, -0.7058231830596924, -0.2020234614610672, 0.8251774907112122, -1.1358493566513062, -0.7679325342178345, -2.7200615406036377, -1.954530119895935, 1.8872215747833252, 0.4353104531764984, 0.8936071991920471, -0.5975635051727295, 0.1299053430557251, -0.5044389963150024, 0.595628559589386, 1.651653528213501, 0.4190027713775635, -1.5776311159133911, 1.3231606483459473, 0.15006029605865479, -1.5434722900390625, 1.8551440238952637, -0.8418397903442383, -0.39805975556373596, -1.0869179964065552, -0.7635732293128967, -0.5078961253166199, -0.5687612891197205, -0.7287503480911255, -1.7842789888381958, 1.4459028244018555, -0.6538563370704651, -0.42299333214759827, -0.5189059376716614, -0.7283146381378174, 0.8552499413490295, -0.005423910915851593, 0.8176307678222656, -0.001522925216704607, 0.44770124554634094, 0.630046546459198, 1.3540912866592407, -0.4229765832424164, -0.8372488617897034, -1.7892460823059082, -0.842587411403656, 0.8068380951881409, 1.3555954694747925, -2.0102550983428955, 1.3788584470748901, 0.04524063691496849, 1.1028691530227661, 0.3190414309501648, 0.3738431930541992, -0.7288128733634949, -1.052411675453186, 2.3682355880737305, -0.5931867361068726, -0.003774631302803755, 1.2856186628341675, -1.9494283199310303, 0.13452252745628357, -2.0098514556884766, -0.22257553040981293, -0.34195953607559204, 0.19755087792873383, 1.707353949546814, 1.3162055015563965, 1.8682198524475098, -0.3832051157951355, 0.6882277727127075, 0.9811802506446838, -0.825284481048584, 1.2099252939224243, 0.041562750935554504, -0.12298446893692017, 1.1022967100143433, 0.07618151605129242, 1.7165038585662842, 0.7448431849479675, -0.4685967266559601, -0.24047407507896423, 0.8400970697402954, 0.22224129736423492, -0.43761134147644043, -0.9070292711257935, -0.6916878819465637, 0.2114514857530594, 0.9870394468307495, -0.5631146430969238, 0.20273597538471222, -0.35651129484176636, -0.5099979639053345, 0.07424169778823853], [-0.3010962903499603, -0.46614667773246765, 0.4956408441066742, 0.27407288551330566, -1.3164693117141724, 0.19439755380153656, 0.3034282326698303, 0.5277153849601746, 0.46814751625061035, 0.08778703212738037, -1.7222414016723633, -0.3070010542869568, -0.7685633897781372, 0.2585074305534363, -0.3000117242336273, -0.675896942615509, 0.058989640325307846, 0.13684935867786407, 1.1471996307373047, -1.245078206062317, 0.405815452337265, 0.9855526685714722, 0.8192386627197266, -0.7133342027664185, -1.1442667245864868, 1.937623143196106, 1.1659502983093262, -1.6901021003723145, -0.5463205575942993, -1.159694790840149, -0.30515116453170776, 1.3154895305633545, 1.660302758216858, 0.737330436706543, 0.4607575833797455, 1.0555351972579956, -2.1979284286499023, 0.5409800410270691, 1.8091903924942017, 0.10920386761426926, -0.5311685800552368, -0.22754447162151337, 1.1605347394943237, 1.4366501569747925, 0.3791131377220154, -1.5560436248779297, 1.5243064165115356, 0.7288411259651184, 1.252277135848999, 0.08313728868961334, 0.9202455282211304, 0.6754865646362305, -1.749108910560608, -0.984633207321167, -1.0015864372253418, 1.536445140838623, -1.4398155212402344, -1.1941258907318115, 0.03981439396739006, 0.015414480119943619, -0.7440212368965149, -2.103327512741089, 1.577272653579712, 0.07873605191707611, 0.4503418803215027, -0.9309035539627075, -1.041459083557129, 0.4926770329475403, -0.43998003005981445, -0.3642413914203644, -0.9508169293403625, 0.7019251585006714, 0.05098960921168327, 0.9268694519996643, 0.16156211495399475, -0.5708790421485901, 2.1622540950775146, 0.678567111492157, 0.37471309304237366, 0.07653550803661346, 0.08653116226196289, 0.5101313591003418, -1.236433982849121, 1.3423662185668945, 0.08858874440193176, -1.7289584875106812, 0.056994710117578506, -0.5809314250946045, -0.3945527970790863, 2.2996344566345215, -1.506795883178711, -0.3692285716533661, 2.250277042388916, -0.32366377115249634, -1.8860348463058472, -0.28781750798225403, 0.4156707227230072, 0.25918954610824585, -0.7264434099197388, 0.25934141874313354, -0.20190294086933136, -0.6562328338623047, 0.6547191739082336, -0.6684677004814148, -0.2761015295982361, 1.6464943885803223, 0.2708318531513214, -0.6646249890327454, -0.4994516372680664, 0.6959385275840759, -0.6760463714599609, -0.054935991764068604, -1.0604888200759888, 0.9361985325813293, 1.2791368961334229, 0.5077879428863525, -1.2103315591812134, 0.3675819933414459, 2.0081779956817627, 0.8196552991867065, 0.3118114471435547, -0.3243366479873657, 0.08788908272981644, -0.0836975947022438, -0.0363430455327034, 0.28388914465904236, -0.5228292942047119, -2.0828161239624023], [-0.5888671278953552, -0.8729358911514282, -0.4420401453971863, 1.496018409729004, -0.7918063402175903, -0.10554894804954529, -0.17484213411808014, -1.3344038724899292, 1.701765775680542, 0.5259021520614624, 1.0764158964157104, 0.09203432500362396, 1.9953902959823608, 0.7917936444282532, 1.559985637664795, 0.1940263956785202, 0.2901041507720947, -0.1778186559677124, 0.3538782596588135, 1.167150855064392, 1.216300368309021, -0.4124109745025635, 0.5275922417640686, -1.6637458801269531, -0.1634204387664795, -0.4843951165676117, -1.5586225986480713, -0.4401068091392517, -0.4073527753353119, -0.13043440878391266, 2.2948808670043945, -0.6675163507461548, -1.2124642133712769, -1.3294360637664795, -1.3262478113174438, -0.6641072034835815, 1.6850780248641968, -0.14632466435432434, 0.26687026023864746, 0.802559494972229, 0.41662630438804626, 1.1845704317092896, -0.35819774866104126, 0.1701115220785141, -0.22882746160030365, 0.6212381720542908, 1.2880555391311646, 1.5538246631622314, 0.24824807047843933, -0.8934218883514404, 1.1223924160003662, 0.5148742198944092, -0.009095367975533009, 0.6478602290153503, -2.0692851543426514, -0.07209726423025131, 0.24255990982055664, 0.2509213089942932, -0.666151762008667, -0.5246931314468384, -0.371567964553833, 1.0374728441238403, -0.7182652950286865, -0.22505757212638855, 1.4948738813400269, -0.5486703515052795, 0.4892028570175171, 0.365203857421875, 1.5507398843765259, 0.006544528063386679, -0.1882414072751999, -2.4000351428985596, 0.3263236880302429, 0.1476754993200302, -0.4317486882209778, 0.6981935501098633, -0.3769916296005249, -1.172091007232666, -2.982684850692749, 0.2588653564453125, 0.36746740341186523, -0.28655770421028137, 0.4454348087310791, -0.7792149186134338, -0.14651724696159363, -0.5579913854598999, -1.3069666624069214, -0.6649685502052307, -0.11009860783815384, -1.7592922449111938, 0.25048476457595825, 0.3026067018508911, -0.22010032832622528, -0.7590686082839966, -0.001821076381020248, -0.08339624851942062, -0.5495213270187378, -0.07597128301858902, -0.2250688076019287, -1.2130180597305298, 1.1001142263412476, -1.8404234647750854, -0.490162193775177, 0.03929832577705383, -0.127357617020607, 0.1565132588148117, -2.0096161365509033, -0.1839761584997177, 1.4329277276992798, 0.18319813907146454, -0.4397083818912506, -0.23508985340595245, 0.5133289098739624, 0.5275806784629822, 0.8895952701568604, -2.2354485988616943, 1.5612884759902954, 0.007694180589169264, -0.7139508724212646, -1.723260521888733, 0.227274090051651, -1.0063148736953735, -1.214538812637329, -1.0912752151489258, 0.16226956248283386, 0.5402055382728577, -0.17342695593833923, -0.724185585975647], [-2.2079081535339355, -0.3807559609413147, 0.45007842779159546, -0.2910979688167572, -1.1855963468551636, 1.0085633993148804, -0.6799038648605347, 0.19623416662216187, -0.5091778039932251, -1.2655423879623413, -0.5519734025001526, 0.4783710539340973, 0.08504263311624527, 1.0267082452774048, -0.2865281403064728, 1.8212273120880127, -0.29754117131233215, 1.578844428062439, -0.2150535136461258, -0.00011100304982392117, -1.2012908458709717, 0.3845697343349457, -1.4454984664916992, 0.2608192265033722, 0.7695085406303406, 0.2892296016216278, 1.071548342704773, -1.0170572996139526, 0.10130129009485245, -0.4336565434932709, -0.5265335440635681, -0.4496026337146759, 0.6332242488861084, 1.8436373472213745, 0.3951650559902191, 0.16544099152088165, 2.368156671524048, 2.053928852081299, -0.7157748341560364, -0.7257778644561768, 1.6239759922027588, 0.13507691025733948, 0.22780752182006836, -1.1563202142715454, -0.5472569465637207, 1.2073659896850586, 0.2050878405570984, -0.9754159450531006, 0.9285914301872253, 0.32255738973617554, -0.41284167766571045, -1.513197898864746, 0.11193013191223145, 0.2540435194969177, 1.272853970527649, 1.888141393661499, -0.32374492287635803, -1.337602138519287, -0.2568269670009613, 1.1115719079971313, -0.497997909784317, -0.6059975028038025, -0.9992600083351135, -0.9472508430480957, 0.6952957510948181, 0.13691608607769012, 0.694299578666687, -1.0608958005905151, 0.7519000768661499, 1.794751524925232, -1.152277946472168, -1.2266243696212769, 1.2141258716583252, 0.12204642593860626, 0.45076918601989746, -1.4509340524673462, -0.6705639958381653, 0.6813525557518005, 1.1343539953231812, 0.22452642023563385, -0.14085005223751068, 0.8939555883407593, 1.6161683797836304, -0.028405901044607162, -0.199262335896492, 1.1094679832458496, 0.5325554609298706, 0.22156299650669098, 0.40452203154563904, -0.1741572767496109, -1.2522363662719727, 0.5625611543655396, -0.560184121131897, -1.2084566354751587, -1.1382285356521606, -0.5591832399368286, 0.012152977287769318, 1.372639536857605, 1.1358518600463867, -0.7943999767303467, 0.4414898157119751, -0.6109907031059265, -1.4904123544692993, 0.4871831238269806, -0.1051623746752739, -1.6665287017822266, -0.21605528891086578, -0.09282579272985458, 1.31141197681427, 1.8054507970809937, 0.618587076663971, 0.8571147322654724, 1.0748200416564941, -0.015678871423006058, -0.7231523394584656, 0.526064932346344, 1.4479831457138062, -0.7269131541252136, -0.5904385447502136, 0.8027592301368713, 0.5605431199073792, -0.3213704824447632, 0.8676798939704895, -1.2625797986984253, -0.40551021695137024, 0.44981449842453003, 1.4716053009033203, 0.45881780982017517], [-0.06677567213773727, -0.01744202710688114, -0.10723664611577988, -1.4979093074798584, 0.3827875554561615, 0.9282540082931519, -1.037846565246582, -0.18630437552928925, 0.16565293073654175, 0.010478384792804718, 1.7277624607086182, -0.5138366222381592, 0.6798896193504333, 0.5450999140739441, 0.48693227767944336, -0.8946651220321655, 0.68342125415802, 1.292662501335144, -0.9728311896324158, 0.4136578142642975, -0.028601637110114098, -0.7202025055885315, -0.04345148056745529, 1.0353477001190186, 0.9564947485923767, 0.9665533304214478, -1.2783186435699463, 0.5736688375473022, 0.11905200034379959, -0.17276866734027863, 1.284256935119629, -1.270235300064087, -1.0682367086410522, 0.9910141229629517, -0.21152810752391815, 0.8403179049491882, -0.8129348754882812, 0.2692539691925049, 0.8213557600975037, -1.088407039642334, 0.3517238199710846, 0.7304828763008118, -1.6593209505081177, -1.1996725797653198, 0.9839473366737366, 0.3921339213848114, -0.04730499908328056, -0.5946482419967651, 1.3281807899475098, 0.24861657619476318, -0.2978070080280304, -0.5816799998283386, 0.5018166303634644, -0.18144910037517548, 0.41592514514923096, -0.18009139597415924, 0.7273623943328857, -1.0786572694778442, 1.132278561592102, 1.1008448600769043, 0.4606543183326721, -0.0777672603726387, 0.25123822689056396, -0.4676339030265808, 0.14400143921375275, -0.9530362486839294, -1.316239595413208, -0.8386267423629761, 1.2865638732910156, 0.6568412184715271, 0.46271583437919617, -2.599560499191284, 0.47061049938201904, 1.492289423942566, 0.09583507478237152, -0.013147450052201748, 1.4844427108764648, -1.1096261739730835, -0.48849087953567505, -0.01407461054623127, -1.3559097051620483, 0.5347741842269897, -1.0516424179077148, 2.2654829025268555, 1.244686484336853, 1.1416304111480713, 0.6159576177597046, 0.24105921387672424, -1.0469797849655151, 1.4721542596817017, 0.3221563398838043, -0.27079781889915466, 0.5743092894554138, 0.5100784301757812, 0.9367366433143616, 0.3196581304073334, -0.6089293956756592, -1.404923915863037, -0.8193190097808838, -0.010836458764970303, -0.8734020590782166, 0.99949049949646, -0.01384648121893406, 1.1275640726089478, 0.2852388024330139, -0.3427668511867523, 0.15849906206130981, -0.7565264105796814, -0.14026454091072083, 2.3633644580841064, -0.03389724716544151, 0.4219358563423157, 0.6175246238708496, -0.6572434306144714, 0.1611565351486206, -0.6696351766586304, -1.935756802558899, 0.580730140209198, -1.9577192068099976, 0.28497567772865295, 1.2236930131912231, -1.2921221256256104, -0.4776718318462372, 1.0284701585769653, -0.7503494024276733, -0.7413192391395569, 1.2045224905014038, -0.9597535729408264], [0.6266219019889832, 1.0656695365905762, 0.36497774720191956, -0.19047366082668304, 0.5473300814628601, -0.29977190494537354, -0.6447175741195679, -1.061428427696228, 0.7140775918960571, -0.872035801410675, -0.6439953446388245, -0.17117267847061157, 1.0574275255203247, -0.999853789806366, -1.4910441637039185, -1.8526283502578735, -0.12482475489377975, -0.2489546537399292, 0.6813679933547974, -0.30586668848991394, -0.05576562136411667, -0.23617438971996307, 0.280491441488266, -0.6048162579536438, 0.09689734876155853, 1.231838345527649, 1.743669867515564, -0.5651826858520508, 1.2082020044326782, -1.96120023727417, 0.0237434022128582, -0.9421653747558594, -0.7540273070335388, 0.6483793258666992, 1.0407469272613525, -0.1616635024547577, 1.0563350915908813, -1.2836031913757324, -0.4817582666873932, 0.43739745020866394, -0.8292506337165833, 1.3917055130004883, 0.8807855248451233, 0.7413285374641418, 0.5096424221992493, 0.606616735458374, -0.07978800684213638, -1.3395318984985352, -0.6615533828735352, 0.3182814121246338, 0.05221723020076752, -1.4013330936431885, -0.17086005210876465, 0.7578516602516174, 2.588823080062866, -0.04764535278081894, 0.7411974668502808, 0.7335402965545654, 2.3779406547546387, -0.6090571880340576, 1.0289114713668823, -0.24920623004436493, -0.6296252608299255, 0.9507511258125305, -0.1968667358160019, -1.312722086906433, -0.3923794627189636, 0.7613840103149414, -1.2930505275726318, -0.8121621608734131, 0.23592136800289154, -0.061245180666446686, -2.4875640869140625, 0.0014181340811774135, -0.9676220417022705, -0.3112477660179138, 0.7339283227920532, -0.3455589711666107, -0.46440133452415466, -1.3368477821350098, -1.856539011001587, 1.86139976978302, 1.154090166091919, -0.5402824878692627, 0.14006048440933228, 0.4667230546474457, 0.22477012872695923, 0.15038612484931946, -0.23805056512355804, -0.9614651799201965, 0.6156486868858337, 0.8405546545982361, 1.4407901763916016, 0.22079670429229736, -1.9252779483795166, -0.2424904853105545, 0.7413623332977295, 2.169820785522461, -1.4376018047332764, -0.22117093205451965, -0.6816776990890503, 1.4880560636520386, -0.974273681640625, -0.07676678895950317, 1.187516450881958, -1.2291181087493896, -1.7949941158294678, 0.3095238208770752, 0.48759353160858154, 0.5828558802604675, -1.9878894090652466, 0.5006663799285889, -0.7798920273780823, -1.071959376335144, -0.9094926714897156, 0.24968698620796204, -1.3064703941345215, -0.15667055547237396, -0.7228443026542664, -0.16961412131786346, -0.7278364300727844, 1.5813015699386597, -1.0964865684509277, -0.16138611733913422, -0.2057550698518753, 0.7911387085914612, 0.21809303760528564, -0.9492507576942444], [1.4175885915756226, -0.10484596341848373, 0.19625942409038544, -0.33325889706611633, -0.20199455320835114, 1.2700486183166504, -0.2302960306406021, -0.7044429183006287, 0.0003902561729773879, -0.487418532371521, 2.337419271469116, -0.06674468517303467, 0.40940186381340027, 0.9587745070457458, -0.9218145608901978, 0.9930933713912964, -0.004019904416054487, 1.630460500717163, 0.13218095898628235, 0.8961275219917297, -0.7224218845367432, 0.22675810754299164, -0.4762032926082611, -1.5166385173797607, 0.21736416220664978, -0.27356237173080444, 1.4601889848709106, -0.030164610594511032, 0.796872079372406, 0.2627004086971283, 1.3601405620574951, 1.1292340755462646, 2.8239974975585938, -1.5608432292938232, -1.0628653764724731, 1.6187938451766968, -0.12573428452014923, -0.1903112679719925, -0.7846515774726868, -1.2856930494308472, -0.7259827256202698, -0.4463128447532654, 0.0597073957324028, 0.21043841540813446, -1.2883695363998413, 0.17208324372768402, 0.5086076259613037, -1.311959981918335, 1.4150232076644897, -0.7390036582946777, -0.18472807109355927, 0.37619277834892273, -0.8777397871017456, -2.1012909412384033, -0.5752623677253723, -1.0974596738815308, 1.022752285003662, -0.7961312532424927, -1.3715887069702148, 0.5123457908630371, 1.4500837326049805, -0.6195673942565918, 1.3468928337097168, -0.672116219997406, 1.6220747232437134, -1.1185925006866455, 0.6198204755783081, -0.180057555437088, -0.8875157237052917, -1.415116548538208, -0.6627956628799438, 0.02048593759536743, -0.7790586948394775, 1.1124902963638306, 0.5016106367111206, 0.1385670304298401, -0.3338978588581085, -0.6859645843505859, 0.18198269605636597, -0.7193092107772827, -1.3812849521636963, -1.6751655340194702, -1.0211750268936157, -1.0106004476547241, 0.027420897036790848, 0.4669235944747925, -2.2873528003692627, 1.762010097503662, 2.4762678146362305, 0.655407726764679, 1.2504020929336548, 0.1352405548095703, -1.477731466293335, 2.3952717781066895, -0.07899325340986252, 1.7003341913223267, 0.9609634876251221, 0.33139845728874207, -0.28764915466308594, 0.46147769689559937, -0.3948603570461273, -2.015636920928955, -0.6230033040046692, -1.3818084001541138, -0.10273273289203644, 0.4072955846786499, -0.21699611842632294, -0.11949636787176132, -0.1077282652258873, 2.0407609939575195, -0.355817049741745, -0.4325893819332123, -2.222378969192505, 0.8740999102592468, -2.175060272216797, 0.7587754130363464, 1.7424854040145874, -1.383730173110962, 0.23871468007564545, 1.3646622896194458, 1.6846954822540283, -0.3357376158237457, -0.2418106645345688, 0.42211005091667175, -0.8399908542633057, -0.6456827521324158, 0.10187357664108276, 0.050538454204797745], [-0.7061660289764404, 0.692658007144928, 2.1873090267181396, -1.0849425792694092, -1.513039231300354, -1.5439058542251587, 0.36168810725212097, 0.33640480041503906, -0.22683703899383545, 0.39830124378204346, -0.013602368533611298, -0.0478845089673996, 0.10994712263345718, 0.7870609164237976, 0.6240395307540894, 0.8136334419250488, 0.27176544070243835, -1.63878333568573, 0.25856658816337585, 1.476829171180725, 0.6700780391693115, 0.5001493096351624, -1.011460304260254, -0.5131648778915405, -1.1265138387680054, 1.1004128456115723, 1.6703522205352783, -0.4463617503643036, 1.681113362312317, -2.0044238567352295, -1.7460036277770996, -0.31328845024108887, -0.2905682325363159, 1.0970520973205566, 0.009378768503665924, -0.6050977110862732, 0.8395001888275146, -0.7775804996490479, 2.0737242698669434, 0.5698915123939514, 0.8025562167167664, -1.2021030187606812, -0.7716171145439148, 1.6355934143066406, -0.5884783267974854, -1.0923221111297607, 0.08019671589136124, 0.2722528874874115, 0.08738229423761368, -0.810801088809967, 0.1565750688314438, 0.8182085156440735, 1.447122573852539, 0.3423595726490021, 1.5011662244796753, -0.09632547944784164, -0.3768565058708191, 0.11807267367839813, -1.2003238201141357, 0.32224491238594055, -0.20910589396953583, 0.35243359208106995, 0.43897199630737305, 1.8521747589111328, -0.7321228981018066, 1.5620436668395996, -0.8922773599624634, -0.07105884701013565, -0.1846451759338379, 1.2013781070709229, -0.47683343291282654, -0.09483342617750168, 0.18848063051700592, 1.4654784202575684, -1.3433098793029785, -0.804851770401001, -0.5327048897743225, -0.24124674499034882, -0.018290361389517784, 1.5586930513381958, 1.0953763723373413, 0.13732893764972687, 1.0789469480514526, -0.7147956490516663, -1.6611645221710205, 0.7371875047683716, -0.7229716777801514, -2.1748740673065186, -1.8703550100326538, -0.08749954402446747, 0.6431605815887451, -0.48507896065711975, 0.9907203912734985, 0.5241549015045166, -2.7912817001342773, 1.08497154712677, -1.460076093673706, -0.04265300929546356, 0.3979432284832001, 0.019461292773485184, -1.465254545211792, -0.09925767779350281, -0.5670543313026428, -0.4193342924118042, 1.6943631172180176, -0.806833803653717, 1.610702395439148, 0.5759039521217346, -0.12046480178833008, -0.3763326108455658, -0.8313230872154236, 0.24612706899642944, -1.0959351062774658, 1.2344119548797607, -1.130405306816101, 0.8385875225067139, 2.3932387828826904, 0.8921683430671692, 1.1296570301055908, 0.6281738877296448, -0.4772859215736389, -2.566222906112671, 1.0292766094207764, -0.20150384306907654, -0.2446235567331314, -1.5588260889053345, 0.10837312042713165, 0.5245603322982788]], [[-0.49508288502693176, 1.1328749656677246, 1.3961580991744995, 1.230374813079834, -0.9251841306686401, 0.29293227195739746, -0.0979001373052597, -0.41319185495376587, -2.6353585720062256, 0.4835047125816345, -1.1814334392547607, -0.04407177120447159, -0.33959072828292847, -0.4905139207839966, 0.3091663718223572, 0.8126451373100281, -0.8655591011047363, -1.962278962135315, -1.5411205291748047, 0.12598948180675507, 0.8272179961204529, -1.5545212030410767, -0.9311359524726868, 0.7698749303817749, 0.7102542519569397, -0.7721611261367798, -0.29180586338043213, -1.2942456007003784, 1.126670002937317, -0.007218505721539259, 0.7374448180198669, -0.006188973784446716, -0.8474590182304382, -1.4374948740005493, 0.3043502867221832, -0.39829808473587036, 1.3979525566101074, 1.0890761613845825, -0.4569302499294281, -0.7945294380187988, -0.7869699001312256, -0.37200191617012024, 0.891444206237793, 2.0983407497406006, 2.020233392715454, 0.23546788096427917, 1.457056999206543, 0.1935955137014389, -0.11112222075462341, -0.8258165121078491, -1.0002063512802124, 2.88533091545105, 0.807851254940033, 0.7057927250862122, -0.639349102973938, -0.17732112109661102, 0.29079535603523254, 0.4441201090812683, -0.5541505813598633, -0.8992765545845032, 0.1030532717704773, -0.20933641493320465, 0.7435040473937988, -0.8459417223930359, -0.35049381852149963, -0.722560703754425, -1.9794631004333496, 0.6809599995613098, 0.8093897104263306, 0.1217445433139801, 0.79069584608078, -1.0355039834976196, -1.1283502578735352, -1.5865237712860107, 0.7481391429901123, 1.5504463911056519, 0.5887798070907593, -0.6836836934089661, 0.8930568695068359, 2.286825180053711, -0.3293077349662781, -0.1745598167181015, 0.4756610691547394, 0.15822264552116394, 0.08564150333404541, -1.7511003017425537, -1.0333608388900757, -0.9374071359634399, 0.6729174256324768, -0.4945610761642456, 1.32024347782135, 1.7613744735717773, -0.8616304993629456, -3.01060152053833, 0.6972779631614685, 0.8262752890586853, 0.7143570184707642, 0.75553959608078, 0.8478005528450012, -0.904625654220581, 0.3009980320930481, 1.445870280265808, 0.9707282185554504, -0.6982276439666748, -0.6851001381874084, -0.6838979721069336, -0.11954076588153839, -0.21257276833057404, 1.5642647743225098, -0.8091035485267639, -0.17242421209812164, 1.0082457065582275, 1.04649817943573, 0.9664633870124817, 0.16304869949817657, 0.6702221035957336, -1.1253762245178223, -0.0398918017745018, -1.1111531257629395, 1.4034626483917236, 1.0416926145553589, 1.814111590385437, 0.15791639685630798, 1.41635000705719, 0.6163884997367859, -0.5827382206916809, 0.673616886138916, 0.6523537039756775], [1.7571483850479126, -0.144189715385437, -0.6776203513145447, -0.17584601044654846, 0.2795966565608978, -0.9190607666969299, 1.0173300504684448, -3.4688568115234375, 1.6697285175323486, 2.4152462482452393, 1.2233240604400635, -0.9601015448570251, 0.5593283176422119, -0.9268233776092529, -2.236603260040283, 0.2217058539390564, -0.8439337015151978, -0.9532984495162964, -0.4202956557273865, -2.337122917175293, 0.31862854957580566, -1.1320080757141113, -1.3186218738555908, -0.03268054500222206, 0.8268275260925293, -2.121828079223633, -0.0612921267747879, -0.2946539521217346, 0.6940603256225586, -0.24481092393398285, 1.098279356956482, -0.3171485662460327, 1.3914697170257568, -0.8934106826782227, -0.619810938835144, 1.2626947164535522, -1.4056575298309326, -0.9962124228477478, -1.4350881576538086, -0.07523766905069351, 0.580938994884491, -0.3681109547615051, 0.30440929532051086, -0.17264492809772491, 0.19737370312213898, 2.098860025405884, 1.8800755739212036, 2.0106863975524902, 0.3954111933708191, -0.26052844524383545, -0.7050338387489319, 1.1604186296463013, 1.1832994222640991, -0.41949227452278137, -0.07686903327703476, 0.809237003326416, 1.648395299911499, -0.1955995261669159, -0.5317882895469666, -0.5641043782234192, -1.2557517290115356, -0.3707937002182007, -0.11095424741506577, -0.5631075501441956, 1.258978247642517, -2.010483741760254, -0.23168422281742096, -0.8271847367286682, 1.0196623802185059, 0.6966983675956726, 0.7153303623199463, 0.35724127292633057, -0.4754006266593933, 1.043428897857666, 0.1699417531490326, -0.032644230872392654, 0.4660240113735199, 0.6243302226066589, 1.2058039903640747, 1.4203200340270996, -1.6840680837631226, 1.3189290761947632, -0.6532136797904968, -0.36365845799446106, -1.6930549144744873, 0.6474118232727051, 0.8760015964508057, 0.8550711870193481, -1.265960454940796, 0.8244131803512573, -0.30187463760375977, -0.0006799378315918148, -1.1550017595291138, -0.7449527382850647, 0.6665607690811157, 0.03477977588772774, -0.007314211223274469, -1.4641213417053223, 0.4210645854473114, 0.00919248815625906, 0.19175465404987335, 0.6483851671218872, 0.27472347021102905, -0.09995579719543457, -1.185138463973999, -0.5139800310134888, 1.5182862281799316, -0.3929627239704132, -0.12400279939174652, 2.7615506649017334, 0.4804028272628784, 0.4166841506958008, 1.213782548904419, 0.7854275703430176, 1.3216909170150757, -0.1148398220539093, -0.5806849598884583, -1.0109328031539917, -1.0693364143371582, 0.1418655514717102, 0.3736174702644348, -1.1492340564727783, 2.394333839416504, -0.6646966934204102, -0.741991400718689, 1.2633662223815918, -0.7153457403182983, -0.471780925989151], [0.9511034488677979, -1.4569872617721558, 1.3490591049194336, -0.6238250732421875, 0.7562414407730103, -1.3268437385559082, -0.945874035358429, 2.2153117656707764, -0.7967441082000732, 0.7484697699546814, -0.4545625150203705, 0.3393722176551819, 1.1188063621520996, 0.10277336090803146, -2.6023876667022705, 1.158372402191162, -0.09991560876369476, 1.4398880004882812, -0.1402847170829773, -0.5015721917152405, 1.1197165250778198, 0.06306006014347076, -1.1568506956100464, -0.36892440915107727, 1.0049724578857422, -0.5139506459236145, -0.022154366597533226, -0.4838860034942627, -0.2635878622531891, -0.7709855437278748, 1.3746429681777954, -1.2088758945465088, -0.5633798837661743, 0.340947687625885, 1.3637686967849731, -2.301226854324341, 0.9077404737472534, 0.8576945066452026, -0.7068519592285156, 1.6514500379562378, -1.165951132774353, 0.625430166721344, -0.21929916739463806, -1.0259926319122314, -0.5824302434921265, 0.37984293699264526, 0.07216043025255203, -1.2306245565414429, 2.3227944374084473, -1.767578125, 0.40340274572372437, 0.9785141348838806, 0.33929574489593506, -1.222662091255188, 0.875041663646698, 0.03324573114514351, -1.1791807413101196, -0.7967506647109985, -0.24432401359081268, 1.2266592979431152, -0.8011435866355896, 0.5348514914512634, -0.906343400478363, 0.44585609436035156, 0.35664939880371094, -0.7670162916183472, -0.21392717957496643, 0.958468496799469, -0.8091232180595398, -0.23188787698745728, -1.1761467456817627, 1.7404799461364746, -0.7738348841667175, -0.1870407611131668, -0.6758869290351868, 0.014321337454020977, -0.9687014818191528, 0.6076318025588989, -0.20655867457389832, 1.2715524435043335, 1.9716107845306396, -0.6231895089149475, 0.6589463353157043, -0.16186675429344177, 0.5452919006347656, -0.23628570139408112, 0.33755823969841003, 0.9945696592330933, -0.22524364292621613, 0.3943175673484802, -1.1644705533981323, 1.166796088218689, -1.699620008468628, -0.046821124851703644, -2.5617518424987793, 0.5739532113075256, -0.8560587763786316, -1.2689769268035889, -1.3386651277542114, -0.3294283449649811, 0.17042163014411926, 0.3137146532535553, 0.08467711508274078, -1.1299399137496948, -1.2217106819152832, -0.8239919543266296, 0.35817432403564453, 1.577905297279358, -2.0781118869781494, 0.9623964428901672, 0.16132965683937073, -1.094744086265564, -1.4583083391189575, -0.6195722222328186, 0.21586638689041138, -1.0020179748535156, -0.1814493089914322, 0.3776662349700928, 0.808404266834259, 0.06436147540807724, 0.042994655668735504, -0.3053033649921417, 2.036292791366577, 0.8024787902832031, -1.3657026290893555, -1.1177995204925537, -0.9476826190948486, -0.343612939119339], [-0.9080019593238831, -0.7085014581680298, -0.15942667424678802, -0.44625696539878845, -1.4412832260131836, 1.9980669021606445, 0.695580780506134, 0.6155415177345276, 1.218881368637085, -1.4199354648590088, 1.5778313875198364, 0.054449040442705154, 0.8398698568344116, -0.48610764741897583, -1.2745096683502197, -0.08457837253808975, 0.0033056605607271194, 0.7542585730552673, -0.8710440397262573, -1.0363099575042725, -0.9151302576065063, 0.10975909233093262, -0.1832091361284256, -1.6664921045303345, 0.23530037701129913, -2.0687286853790283, 0.4722689092159271, 0.3910665512084961, -1.2517441511154175, -1.499729871749878, -0.6617249846458435, 1.115363359451294, -0.2432222217321396, 1.6815886497497559, -0.5591974258422852, -0.3667178750038147, -0.6577770709991455, 0.10294327884912491, 0.3034133017063141, -0.08117537200450897, 0.34931281208992004, 0.3884942829608917, -0.3936682641506195, -0.17506031692028046, 0.9595673084259033, 1.412243366241455, 1.756700873374939, 1.5303925275802612, 0.9736753702163696, 0.8201487064361572, -1.1878389120101929, 1.278456449508667, -1.2469228506088257, -0.27679717540740967, -0.4788593351840973, 1.7245601415634155, 0.6796795725822449, -1.2356212139129639, 0.28026282787323, 0.8354426026344299, 0.7629198431968689, -0.6150855422019958, -1.1926828622817993, -0.755844235420227, 1.3208881616592407, 0.7383714914321899, -0.14667369425296783, -1.8577266931533813, 0.7304806709289551, -0.9790018200874329, -1.617798089981079, 0.1672177016735077, 1.0009108781814575, 0.5794539451599121, -1.3770487308502197, -1.1279523372650146, -1.3636163473129272, 0.5770522952079773, -0.9083077907562256, -0.9344663023948669, -0.25176289677619934, -1.9177168607711792, 0.9162348508834839, 0.23533038794994354, 0.895155131816864, 0.7564346194267273, 0.16169056296348572, 0.8617466688156128, -1.5111836194992065, 1.664398193359375, -1.153154730796814, 1.2884845733642578, -1.5107667446136475, 0.18582570552825928, -1.2223246097564697, 0.3587959110736847, 1.1661783456802368, -1.63419771194458, -1.3049312829971313, 1.0147334337234497, -0.9336094260215759, 1.0767297744750977, 0.8424339294433594, -0.785117506980896, -1.1395004987716675, -0.1723351925611496, -1.6675608158111572, -1.2348406314849854, -0.7174457311630249, 0.9553810358047485, 0.4484078288078308, -1.550743579864502, -2.021486759185791, 0.8234588503837585, -0.24075621366500854, -1.231345772743225, -0.06817077845335007, 0.6003772616386414, -1.1422688961029053, -1.1059073209762573, 0.2755086421966553, 0.6136115789413452, 0.28083452582359314, 1.5083647966384888, 0.8127315640449524, 1.0477194786071777, -0.3986939787864685, -1.0410557985305786], [-0.5100918412208557, 0.6232295632362366, 1.8086851835250854, 0.058501169085502625, -0.30232101678848267, -0.48470038175582886, -0.1976652592420578, -0.15354984998703003, -1.2580602169036865, 0.3079884648323059, -0.7354320883750916, 1.2649877071380615, 0.8330971598625183, -0.11703111231327057, -0.2380637377500534, -0.03434395045042038, -1.3172415494918823, -1.3996386528015137, 0.09636420756578445, -1.3303399085998535, 2.157578229904175, 0.6618550419807434, 1.7335355281829834, 0.0639161765575409, 0.33747774362564087, -1.090152621269226, -1.674756407737732, 0.9152576327323914, 1.9170223474502563, -0.5251891016960144, 0.624060332775116, 1.8611258268356323, 1.8477801084518433, -0.5633484721183777, 1.4938558340072632, 0.40963107347488403, 0.1846439689397812, 0.8825927972793579, 0.14180110394954681, -1.6132296323776245, 0.5788057446479797, 0.027845382690429688, 0.24108076095581055, -0.9677678346633911, -0.0036369096487760544, 1.2421290874481201, -0.5571635365486145, -0.3537626564502716, 0.3417311906814575, 0.889712929725647, 0.876240611076355, 0.40528053045272827, 0.7705699801445007, -0.8169817328453064, 1.4448463916778564, -0.3113642632961273, -1.217882513999939, -0.19266735017299652, -0.6672942042350769, 0.22888197004795074, 0.015578516758978367, -0.6969597339630127, 0.48042187094688416, 0.24925589561462402, 1.602264642715454, 0.27160996198654175, 0.4291018545627594, -1.6239728927612305, -1.0845803022384644, 0.8300182223320007, 0.4208700954914093, -1.2491097450256348, -1.5166501998901367, 0.5675206780433655, 0.16243787109851837, -0.3032681345939636, 0.31181979179382324, -0.9413207769393921, 0.48224684596061707, -0.26577889919281006, 2.314321279525757, -0.07155791670084, 0.719210684299469, -0.916061282157898, 0.49141359329223633, -0.0997769683599472, 0.68853759765625, -0.6569565534591675, -1.3604761362075806, 0.5333590507507324, -1.0769362449645996, -0.16720744967460632, 0.9936308264732361, 0.08390196412801743, -0.3452625274658203, -0.2585723102092743, 1.688887119293213, 0.404936283826828, -0.2224532514810562, 0.11664694547653198, 1.193724274635315, -0.845902144908905, 0.7335996627807617, -1.3365790843963623, -1.1822830438613892, 0.5146400928497314, -0.5787268280982971, 0.7504107356071472, -0.4608525037765503, 0.137781023979187, 1.0588897466659546, -1.3820810317993164, 0.4818870723247528, 0.9554759860038757, 0.2660694420337677, 1.5634238719940186, 1.8406100273132324, 0.3103412091732025, 0.40955406427383423, -1.7485411167144775, -2.490184783935547, -0.4006803333759308, 1.8000091314315796, -1.8723863363265991, -0.37754878401756287, 1.3213671445846558, 0.007028850261121988, -0.46481066942214966], [-0.32186463475227356, -0.03967634215950966, 0.2687079906463623, 0.9532909989356995, 1.8114705085754395, -1.734514594078064, 2.271930694580078, 0.06984341889619827, 0.5138711929321289, 0.08573366701602936, 0.09295016527175903, 0.6472401022911072, 0.6489006876945496, 0.681633472442627, 1.5458064079284668, -0.0970909371972084, -0.08495450019836426, 1.814531683921814, -1.7926243543624878, 0.39246806502342224, 0.6539990305900574, -0.30611011385917664, -0.35094743967056274, -0.9589783549308777, 0.6857471466064453, -0.927910566329956, -0.6147103309631348, 0.5748581886291504, -0.458061158657074, 0.7877654433250427, -0.5384759306907654, -1.1874891519546509, 0.4493350684642792, 0.6752777695655823, 0.5707970857620239, -1.4812825918197632, -0.08675703406333923, -0.9255431294441223, -0.2040042132139206, 0.18977896869182587, -1.5580581426620483, -1.841045618057251, 0.21348842978477478, 1.0965899229049683, 1.5269832611083984, -0.11534105986356735, 0.971604585647583, -0.5265868902206421, 0.7122147679328918, 1.0948179960250854, 0.4449608325958252, 1.4941643476486206, 0.7084054946899414, -0.45236706733703613, 1.0605899095535278, -1.3721188306808472, -0.7546606063842773, -0.28813499212265015, 1.6900383234024048, -0.3709995150566101, -0.28127819299697876, -0.9919174313545227, -1.5861835479736328, -0.17898234724998474, 1.5788121223449707, 0.6694008111953735, -0.1745292693376541, 0.38342100381851196, 0.5415528416633606, -0.4322778880596161, 0.7675319314002991, 0.6963223218917847, -0.2946287989616394, 0.8865789175033569, -0.38086870312690735, 1.7134145498275757, 0.4869231581687927, -0.701119065284729, -0.3124637305736542, -0.6361902952194214, -0.8836224675178528, -0.25512775778770447, -0.7839655876159668, 0.9924608469009399, 1.7210819721221924, 0.3123651146888733, 1.613970398902893, -0.956985592842102, -0.5637637972831726, 1.3757643699645996, -0.18901710212230682, 1.8050817251205444, -0.6918593645095825, 0.3659283220767975, -1.3524326086044312, 0.7275193333625793, 0.33131372928619385, 1.201829195022583, -0.23585999011993408, 1.8236713409423828, 1.3162460327148438, 0.029693076387047768, 0.6687415242195129, -0.24559548497200012, 0.7651791572570801, -0.9464155435562134, 0.0265171118080616, 2.7770628929138184, 1.464986801147461, -1.3254687786102295, -0.6006187796592712, 1.1997535228729248, 0.013212078250944614, -0.1837039291858673, -0.7462632060050964, 0.45851871371269226, -0.016358910128474236, -0.21670542657375336, 1.2141326665878296, 0.3329261541366577, -2.2432668209075928, -0.3270530700683594, -1.0930564403533936, 0.2543773651123047, 0.3539987802505493, 0.012547356076538563, -0.16854654252529144, 0.2511204183101654], [-0.08623301237821579, -0.7500137686729431, -0.0163438618183136, 0.2569428086280823, -0.9930922985076904, 1.041405439376831, -0.33061373233795166, 0.41478976607322693, 1.2213695049285889, 0.5913141965866089, -1.4720873832702637, 0.028427133336663246, 0.14660370349884033, -0.5930120944976807, -1.216372013092041, -0.21267813444137573, 2.1859066486358643, -0.16824203729629517, 0.9048829078674316, 0.5533968210220337, -1.1170618534088135, -0.8189292550086975, -0.47958359122276306, 0.15114712715148926, -0.053280167281627655, -0.7524537444114685, -0.20233575999736786, 0.6861377358436584, 1.4037141799926758, -0.08263858407735825, 0.16898193955421448, -1.0891107320785522, -0.6220728754997253, -1.5963473320007324, 1.2077999114990234, -1.3852849006652832, 0.35383254289627075, 0.6661660075187683, -1.3632482290267944, -0.1844484955072403, -1.2887412309646606, 0.8529197573661804, -1.0487139225006104, -0.4399177134037018, 0.36342063546180725, -0.02047405205667019, -0.5313677191734314, -1.0974773168563843, -1.3010878562927246, 0.18668042123317719, 0.064715176820755, -1.6962802410125732, -0.3133244812488556, 0.6063781380653381, 1.4275840520858765, 0.6745781898498535, -0.2623496651649475, -0.13037893176078796, -0.6610027551651001, -2.2970175743103027, -0.34139761328697205, 0.46033549308776855, 1.4946037530899048, -0.051780279725790024, -1.4767026901245117, -0.9026824235916138, 0.6896226406097412, 0.3618784248828888, 0.8117616772651672, 0.6553581357002258, 0.3668031394481659, 0.6174401044845581, -1.882096767425537, -0.21411144733428955, 0.25653940439224243, 0.025592559948563576, -0.7930683493614197, 2.2063491344451904, -1.0505633354187012, -1.073182463645935, 0.2952166497707367, -1.490604281425476, 1.5861715078353882, 0.16269521415233612, -1.7811853885650635, -2.182309627532959, 0.728000819683075, -0.38749220967292786, -0.9117146134376526, -2.7906105518341064, 1.744601845741272, -1.3145548105239868, -0.38897252082824707, -1.0951672792434692, 0.5989730358123779, -1.6974537372589111, 0.9237443208694458, 0.29999998211860657, -0.05933864042162895, -1.6104655265808105, 1.176109790802002, -1.1296696662902832, 0.6842830181121826, 0.22723886370658875, 0.10922832041978836, -1.577289342880249, -1.0140552520751953, 0.8389034271240234, 0.9061703681945801, 0.14515255391597748, 1.025789499282837, 0.3279099464416504, -1.2837504148483276, -0.46282505989074707, -0.08283045887947083, 1.1198683977127075, 0.91656494140625, -0.6921628713607788, 0.6225461363792419, 0.905899703502655, 1.0117751359939575, -0.32295775413513184, 0.6832431554794312, -2.2285263538360596, -0.17586474120616913, 1.1382993459701538, -0.760958731174469, -0.029910411685705185], [0.7508143782615662, 0.8804890513420105, 0.6675572395324707, 1.2077977657318115, 0.4600943624973297, 0.9378392696380615, -1.9473812580108643, -1.1537322998046875, -0.16474321484565735, 1.1277580261230469, 2.4442431926727295, -0.9367775321006775, -0.9721737504005432, -0.4643744230270386, -0.619594931602478, -0.7061487436294556, -0.15942442417144775, -0.8670228719711304, 0.01695151813328266, -1.510236382484436, -0.18283408880233765, 1.3224706649780273, -0.245241180062294, -0.7640751004219055, -0.47562775015830994, 1.4019283056259155, -0.053708963096141815, -2.2001841068267822, -0.6359044909477234, 0.044209886342287064, 1.811004877090454, -0.9908022880554199, 0.3538617789745331, 1.327064037322998, 0.056219737976789474, 0.27827394008636475, -0.3237343430519104, -0.3963966965675354, 0.9232738614082336, -0.27375495433807373, 0.6500082612037659, -0.4978249669075012, -0.3089657127857208, 0.5357304811477661, -0.5490400791168213, 0.4315902888774872, -1.9481275081634521, -0.8887869119644165, -1.7243211269378662, 0.07717035710811615, -0.09774759411811829, -0.5544987916946411, 0.6591371297836304, 0.17157045006752014, 0.8075145483016968, 0.8738911747932434, 0.6165876388549805, 1.4331296682357788, 1.6303634643554688, -0.9878137707710266, 0.07394150644540787, -0.25234997272491455, 1.0088897943496704, 1.024867057800293, 0.6765250563621521, 1.6195272207260132, 0.8306647539138794, -0.07609399408102036, 0.9082561731338501, -2.1670937538146973, -0.45381927490234375, 0.4640326499938965, 1.0024744272232056, -0.21473892033100128, -0.4924451410770416, -0.4820663034915924, -1.4255971908569336, -1.2597631216049194, -0.6495846509933472, -1.4964900016784668, -0.4704555571079254, 0.8134022951126099, -1.7512692213058472, 2.165767192840576, -0.7120457291603088, 2.561640501022339, -0.6157405376434326, -0.0510292686522007, 0.36992284655570984, -0.8233150243759155, -1.0189552307128906, -0.7379257082939148, 1.9517027139663696, 0.7233820557594299, 0.6804953813552856, 1.6275087594985962, -0.7661633491516113, 0.8695118427276611, -1.3984888792037964, -0.4666454792022705, -0.9323054552078247, -0.7259492874145508, 0.27547773718833923, -1.077200174331665, -0.4127342700958252, 1.995727300643921, -1.7663202285766602, 0.820022702217102, -1.7676922082901, 0.867838442325592, 0.7637401819229126, -0.6747188568115234, 0.35135889053344727, -0.18159155547618866, -1.3461503982543945, 0.5745491981506348, -2.842054605484009, 0.1438969075679779, -1.952036738395691, 0.7283703684806824, -0.45431262254714966, -1.3630340099334717, -0.3836321532726288, 0.27062252163887024, -0.14044919610023499, 0.8798125982284546, 0.043228425085544586, -0.35047706961631775], [0.09243419766426086, -0.19719786942005157, 0.35526973009109497, 0.2903487980365753, 0.5528693199157715, -0.6956190466880798, 0.2504681944847107, -0.05103857442736626, -1.3633816242218018, -1.2183830738067627, 0.6730000972747803, -0.9549188613891602, 0.5841822028160095, -0.6956800222396851, 0.17459756135940552, 0.717405378818512, -0.746630072593689, 1.265425682067871, -0.3684048354625702, -0.40494999289512634, -1.1012694835662842, 0.08781357109546661, -1.2683628797531128, 0.4143211245536804, 1.0672417879104614, -0.033794231712818146, 0.2660815417766571, -0.46429818868637085, 0.033290181308984756, 1.3110535144805908, 0.3958507180213928, 0.9012513160705566, -0.6596696972846985, 1.2469475269317627, -1.0979597568511963, -0.5866738557815552, -0.908618152141571, 1.895112156867981, 1.4237335920333862, -0.41727033257484436, -0.2879008948802948, 0.08141497522592545, 1.860302448272705, 2.1016297340393066, 0.7336816787719727, -1.0851929187774658, 2.319361686706543, 0.2436305284500122, 0.11921555548906326, -0.11179198324680328, -0.5629292130470276, -1.2131465673446655, -0.8960766196250916, 0.17367786169052124, 1.1985284090042114, -1.676523208618164, 0.5584713816642761, 0.5194028615951538, 0.3474707305431366, 1.1561297178268433, 0.7662968039512634, 1.1111806631088257, 1.0788037776947021, 0.6233105659484863, -0.3051409423351288, -0.8988713622093201, 0.24259339272975922, 1.6303383111953735, 0.6146303415298462, 1.5490567684173584, 0.9459452033042908, 0.6571423411369324, -1.3454793691635132, 0.8886927962303162, -0.5446719527244568, -0.38791152834892273, -2.4319112300872803, -0.40773236751556396, -1.1748136281967163, 0.2858582139015198, -0.17120547592639923, -0.0790870264172554, 0.15939827263355255, 0.10829578340053558, -1.1894711256027222, -0.2978963255882263, 0.6899535655975342, -1.0167099237442017, -0.7325149774551392, 0.11312716454267502, -0.20729592442512512, 0.002055802382528782, -0.4838242828845978, 1.5647599697113037, -2.0796759128570557, 0.7125909328460693, -0.1912710815668106, 0.6761711239814758, -0.20555053651332855, -0.11007510870695114, 0.7637658715248108, -0.3933955132961273, -1.686410903930664, 1.292258858680725, 1.0592433214187622, 1.2743661403656006, -2.1640541553497314, 0.4489961564540863, -0.475034236907959, -0.7244725227355957, 1.2118444442749023, -1.178162932395935, 1.4564638137817383, -1.3166654109954834, -1.0927317142486572, -0.850770115852356, 2.2462801933288574, 0.3514675796031952, -1.0748573541641235, 1.156882405281067, -0.25487250089645386, 1.7231451272964478, 0.4991452097892761, -1.4501631259918213, 0.8187306523323059, -0.6427155137062073, 0.1880887895822525, -1.6650077104568481], [0.6861661672592163, 1.1586965322494507, -0.9540385603904724, -0.1030685305595398, 1.808287501335144, -0.3017609417438507, -0.2780129015445709, 0.5054608583450317, 0.3637795150279999, -0.7000718116760254, -1.3082404136657715, -1.3744256496429443, 0.4589519798755646, -1.1561691761016846, -2.1790332794189453, -0.11086831241846085, 0.14987094700336456, 0.9963992834091187, 1.4258062839508057, 0.7011227607727051, 1.310156226158142, -0.7431901693344116, -1.2558366060256958, 0.023980015888810158, -0.834385335445404, -0.3425386846065521, -0.37717390060424805, -0.5936740040779114, 0.48807549476623535, 0.9901614785194397, 0.06557508558034897, -1.3909579515457153, 0.28357672691345215, 0.38667401671409607, 0.09997442364692688, -0.5870243906974792, 1.423701524734497, 0.6191157102584839, 0.001158414175733924, 0.07595094293355942, -1.1847572326660156, -0.831809401512146, 0.3220081329345703, 0.07360544055700302, -0.9255780577659607, 0.17337119579315186, -0.112961046397686, 0.13435505330562592, 1.9877710342407227, 0.3874928653240204, -0.720302402973175, -1.1336402893066406, 0.6234480738639832, 1.3270162343978882, 0.7199123501777649, -1.7548953294754028, 1.5833368301391602, 2.444082021713257, 0.2530279755592346, -0.5966771245002747, -0.7220675349235535, 0.2278953343629837, 1.4989172220230103, 1.1962777376174927, 1.7826393842697144, 0.13600073754787445, -1.3209055662155151, 1.5678257942199707, -2.5782148838043213, -1.0312340259552002, 0.8931007385253906, -2.0318009853363037, -1.6357457637786865, -0.5567817687988281, -1.4498664140701294, 1.7784401178359985, -0.4506191909313202, 1.6513590812683105, 0.13584622740745544, -0.47538861632347107, -0.06190406531095505, 0.2965027093887329, -0.3547075688838959, 1.2090433835983276, -0.32656362652778625, -0.8842399716377258, -1.2870761156082153, -0.8161007165908813, 0.13445818424224854, 1.7426650524139404, 0.5717208385467529, 0.3058224022388458, -0.2291405349969864, 0.11176743358373642, -0.2636219263076782, 0.39001065492630005, 1.2446072101593018, -1.3898266553878784, -2.3219974040985107, -0.13376645743846893, 1.158827543258667, -1.0453041791915894, 0.9944304823875427, 0.8040704131126404, 1.0886515378952026, -1.4835057258605957, -0.29238077998161316, 0.0662156417965889, -0.20581039786338806, -1.2875442504882812, -0.6347774863243103, -0.1514122635126114, -0.32482677698135376, -0.3229949176311493, 0.3071039319038391, -0.3714844882488251, -0.9206189513206482, -0.18909025192260742, -0.902641236782074, -0.9475052952766418, 0.5181182026863098, -0.18640758097171783, 0.4773140847682953, -2.128268241882324, -1.649412989616394, 0.34098920226097107, -0.6843554973602295, -0.5793932676315308], [-1.4107974767684937, 1.164811372756958, 0.7825561165809631, 0.46355417370796204, 0.07010312378406525, 0.8636038899421692, 1.4895273447036743, -1.1994339227676392, 1.456946611404419, -1.2536201477050781, -0.9576086401939392, -1.507629156112671, -0.7479499578475952, -1.060295581817627, 0.9322208166122437, 0.44512873888015747, 1.8490116596221924, 0.04864639416337013, 0.5610709190368652, 0.9775897860527039, -0.1915052831172943, 0.8026541471481323, 0.6198790073394775, -0.5108618140220642, 1.4570748805999756, 2.3536465167999268, -0.0713043063879013, 0.1421128362417221, 0.4584019184112549, 0.10341149568557739, 0.4548494517803192, 0.9704174995422363, 1.008022665977478, -1.49393892288208, -0.6404075622558594, -0.06996802240610123, -0.5878878235816956, 0.7541680932044983, -0.28572705388069153, 0.08310143649578094, 0.10198412090539932, -1.4311068058013916, 0.16659654676914215, 0.4616917669773102, -0.28317826986312866, 2.661534547805786, -1.7532397508621216, -0.018531406298279762, 1.03917396068573, 0.4408700466156006, -0.15750063955783844, 1.0936970710754395, 1.0546640157699585, -0.5418999195098877, 1.8543590307235718, -0.572762131690979, 0.5438879728317261, -0.04090338200330734, 1.5397543907165527, 2.3695735931396484, 1.4170085191726685, -1.197073221206665, 1.37533700466156, -0.6519966721534729, -0.2713627219200134, -0.949390172958374, 1.500816822052002, -0.6088547110557556, 1.5969139337539673, -0.27027103304862976, -1.850471019744873, -0.49589410424232483, -1.1700470447540283, 2.2715036869049072, -1.314210057258606, 0.8080595135688782, 1.006608247756958, -0.5224975943565369, -0.014940634369850159, -0.7134578227996826, 0.18429236114025116, 0.355689138174057, -1.146065592765808, -1.8783930540084839, 0.1423303335905075, -0.3044191002845764, 0.35446953773498535, -0.2886090874671936, 1.0927938222885132, -0.9498319625854492, 0.3493765592575073, -0.16784512996673584, 0.2237645536661148, 0.2396814078092575, -0.7843520045280457, 0.4843965172767639, 2.8160958290100098, 0.16600815951824188, 0.16137798130512238, 3.119450569152832, 0.8604443073272705, -1.633481502532959, -1.3623262643814087, 0.5767697691917419, 1.7226760387420654, 0.29744774103164673, -1.2431000471115112, 0.6141732335090637, -1.0467907190322876, 0.8161552548408508, 0.8506627082824707, -0.2407366931438446, -1.0049684047698975, -1.299586296081543, -1.1480224132537842, 0.8401852250099182, -0.7174710035324097, -0.35106512904167175, -1.671422004699707, 0.4103314280509949, 2.6671302318573, 0.49758535623550415, -0.18691237270832062, 0.30552029609680176, -1.0337241888046265, -0.07174956798553467, -0.6159740090370178, 0.48334774374961853], [0.3120023012161255, 0.2064107060432434, 0.29493582248687744, -0.2830115258693695, 0.8840492963790894, 0.753923237323761, -0.595608651638031, 0.03425909951329231, -0.2604927122592926, 0.9612552523612976, 1.8317197561264038, -1.2351499795913696, -0.7543461918830872, 0.4274824261665344, -0.7655925154685974, 1.5265636444091797, -2.6210954189300537, -0.8450833559036255, -0.8019245862960815, 0.2128680944442749, 1.2948946952819824, 0.855659544467926, -2.3721747398376465, 2.0434763431549072, 0.9966643452644348, 0.025749558582901955, 1.3812825679779053, 0.06953930109739304, -0.5126495361328125, 1.2516350746154785, 0.14927557110786438, 0.8419896364212036, -0.5065218806266785, -0.12951351702213287, -0.7204043865203857, 0.7494507431983948, 0.22513625025749207, 1.140165090560913, 0.1405654400587082, 1.4386134147644043, 0.09486936032772064, 1.6417573690414429, 0.2624863386154175, 0.20023934543132782, 0.606215238571167, 0.5001251101493835, -0.03658226504921913, 0.9809300899505615, 0.5470807552337646, 0.2765684723854065, -0.35999795794487, 0.5465337634086609, -1.3668169975280762, 0.3330184519290924, 2.1022274494171143, -1.0130921602249146, -0.16304410994052887, 2.189793825149536, -0.24048694968223572, 1.6812217235565186, 0.8208482265472412, 0.8831862211227417, -0.4664374887943268, -1.0357345342636108, -0.6045982837677002, 0.7480955719947815, -0.42538729310035706, 0.7345123887062073, -1.0520159006118774, 0.17982704937458038, -0.09401337057352066, 0.7049108743667603, -0.6491562724113464, -1.1253000497817993, 0.3104458749294281, -0.43350112438201904, 0.9214242696762085, 1.8117469549179077, -0.18384012579917908, -0.03366899490356445, -1.429124116897583, -0.23894871771335602, 0.7363168597221375, 0.7775614857673645, 0.34998640418052673, 0.7101022005081177, -1.3948842287063599, 1.977696418762207, 0.2538602650165558, 0.4166052043437958, 1.3424550294876099, 0.1385132372379303, -1.2144153118133545, -1.1494771242141724, 0.36458534002304077, 0.642536997795105, -0.764325737953186, -0.19508102536201477, 0.39220571517944336, 0.04149281978607178, -0.176932230591774, -0.4349747598171234, 0.14064154028892517, 1.2106163501739502, 0.44444870948791504, 1.5961788892745972, 0.12734083831310272, 0.9582749605178833, -0.16429343819618225, -0.27679699659347534, 1.0377057790756226, -0.1590445190668106, -0.7002177834510803, 0.2969692349433899, -1.2177362442016602, -0.425715833902359, 0.5435605049133301, 1.606937050819397, -0.24453826248645782, -0.6394344568252563, -1.3051056861877441, -0.5060433149337769, 0.7018314599990845, 0.8343087434768677, -0.3598111569881439, -0.3011554181575775, -0.803534209728241, -0.03841876611113548], [-1.786266565322876, 0.9422690272331238, 0.2611129581928253, 0.6461262702941895, -0.12669146060943604, 1.872568964958191, 1.181038498878479, -0.39825621247291565, -0.26362836360931396, 0.12090317159891129, -0.2831224501132965, -0.05526662990450859, 0.8248088359832764, -0.02585688605904579, -0.22020787000656128, -0.15157467126846313, -0.35861366987228394, -0.6345399618148804, 0.48920825123786926, -0.19624224305152893, -0.09130654484033585, -0.6993352770805359, 0.06492653489112854, -0.9411167502403259, -0.2739810645580292, -2.106375217437744, 0.7924509048461914, -1.8782037496566772, 0.6039733290672302, 0.8149853348731995, 0.9372628331184387, -0.8463279008865356, -1.9299486875534058, -0.9522243142127991, -2.0212621688842773, -0.9684590697288513, -0.1959788203239441, 0.1452917754650116, -0.5905547142028809, -0.09878353029489517, 0.6940068602561951, -1.3569960594177246, 1.2762155532836914, 1.9657635688781738, 1.5696275234222412, -0.5842763781547546, 0.6829372048377991, 0.10567888617515564, -0.7485142350196838, 1.4209479093551636, -1.6028305292129517, -0.6733203530311584, -1.2114460468292236, -0.27000319957733154, -1.3823527097702026, -1.1288121938705444, 2.08378005027771, 0.6811627745628357, 0.5481606125831604, 0.16044622659683228, 0.15690232813358307, 0.04397150129079819, -0.813666582107544, 1.418843388557434, -0.593859851360321, -1.933053731918335, 0.7024792432785034, -0.8400307297706604, 0.6020234823226929, 1.3559948205947876, -0.09583178907632828, 0.38540366291999817, -0.6242464184761047, -0.10865715146064758, 0.18497389554977417, 1.3539108037948608, -1.0874825716018677, 0.9859365820884705, 0.5471537709236145, 0.37114012241363525, -0.8034869432449341, 1.0677852630615234, -0.9391744136810303, 0.42515406012535095, -0.8655396103858948, 1.7993487119674683, 0.04785827919840813, -0.029730115085840225, -2.146721839904785, 1.3640713691711426, 0.33150655031204224, 1.1739308834075928, 0.520731508731842, -0.5572770833969116, -1.6464297771453857, 0.01840073987841606, 1.8839735984802246, 0.3009081780910492, -1.8000757694244385, 1.695286750793457, 0.752435028553009, -0.25293949246406555, -1.3529070615768433, 0.8457263112068176, -1.1691657304763794, 0.04808575287461281, -1.1345889568328857, 0.6738744378089905, 0.4907242953777313, 0.5237848162651062, 0.44562551379203796, 0.07224800437688828, -0.8609312176704407, -0.333584189414978, 0.07678785175085068, 0.621846616268158, 0.26409822702407837, 0.7978019118309021, 0.5159680843353271, -1.0419379472732544, -0.07952912896871567, -1.8178967237472534, 1.6402946710586548, 0.12395127862691879, 0.30017006397247314, -3.191709041595459, -0.0181479062885046, -1.0754257440567017], [0.672335684299469, -0.04377516731619835, -0.34329742193222046, -0.3072390556335449, 0.09225436300039291, 0.5074530243873596, -0.3907258212566376, 0.25382986664772034, -0.3730001747608185, 2.1949973106384277, -0.819499671459198, -1.7539777755737305, -1.3954998254776, 2.0775957107543945, 0.7154034972190857, 0.6953958868980408, 0.9925193190574646, 0.4601011872291565, 0.2177465409040451, -1.5876617431640625, -1.6456406116485596, -1.609087347984314, -0.24238064885139465, -1.3544007539749146, -0.8736339807510376, -0.3910912871360779, -1.9267492294311523, -0.5628005266189575, 1.715964913368225, -0.07891280204057693, 0.18601514399051666, -0.7478120923042297, -0.7459470629692078, -0.9567577838897705, 1.6340951919555664, -0.2509170472621918, 0.8623403310775757, 1.8004413843154907, -0.45957034826278687, 0.3973775804042816, 0.3614029586315155, -1.0425692796707153, -0.4164256751537323, 1.4014471769332886, 0.666182816028595, 0.19344884157180786, 1.0144710540771484, -0.37174832820892334, 0.8597614169120789, -0.7369722723960876, -0.5408207178115845, 0.9447195529937744, 1.2774417400360107, 0.06515564769506454, -0.46176984906196594, -1.7491803169250488, 0.5269985198974609, -1.2226873636245728, -0.3106762766838074, 1.0215232372283936, 0.9906831979751587, -0.3385036289691925, -0.6592729687690735, -0.2666237950325012, -1.0514658689498901, 1.7141547203063965, -0.47334524989128113, -0.9010960459709167, 1.2377876043319702, 0.2921079695224762, -0.23698623478412628, 1.0572054386138916, -2.2796530723571777, -1.7405815124511719, -0.07410617917776108, 0.4980258047580719, -0.47794243693351746, 0.4893878102302551, -0.2935083508491516, -2.6959989070892334, -1.6009671688079834, -1.5643786191940308, 0.1740335077047348, -0.37354007363319397, -0.23932628333568573, -0.755660891532898, 1.3618770837783813, 0.6806793212890625, 0.4570806920528412, -0.6017734408378601, -1.001293659210205, -0.3380434811115265, 0.6417784094810486, 0.4487229585647583, 0.5366929769515991, 0.07766091823577881, 0.043082885444164276, 2.5309550762176514, -1.4203678369522095, 0.7315963506698608, 0.8892641663551331, -0.16202369332313538, -0.28723058104515076, -0.908298134803772, 2.2363548278808594, 0.38736504316329956, 0.8021628260612488, -0.3467722237110138, -1.2663459777832031, 1.3242645263671875, -1.6417566537857056, 0.9983874559402466, -2.605699300765991, -0.39235395193099976, -0.5969581604003906, -0.5757119655609131, 0.4330138564109802, 0.9880099296569824, -0.030738120898604393, 0.05619833990931511, 0.3564167618751526, 0.3994365930557251, -1.1573485136032104, 0.294023722410202, -2.5547072887420654, 2.3312201499938965, 0.29071348905563354, -0.6317571401596069], [0.3920152485370636, 0.08491010963916779, 0.14989131689071655, -0.11895479261875153, -0.7717357873916626, -1.6934568881988525, -1.3371453285217285, -0.1053435355424881, 0.815041720867157, 0.6734583377838135, -0.7124800086021423, 0.18746784329414368, 1.8948349952697754, -0.9340279698371887, 0.03898165002465248, -0.5244053602218628, -1.7038135528564453, 0.5558773279190063, 0.8169310092926025, 0.4527648687362671, -1.6455756425857544, -1.0116984844207764, 0.40506768226623535, -0.7635840177536011, 0.33487385511398315, 0.7284679412841797, -0.5479484796524048, 0.464743971824646, 0.9272478818893433, -0.35394221544265747, 0.48869025707244873, 0.9053283333778381, 0.6641244888305664, 0.773732602596283, -0.658649206161499, 1.677804708480835, 0.499105840921402, 0.6513407230377197, -0.3214348256587982, 0.5161388516426086, -0.6087324023246765, -0.5731064677238464, -0.5818008184432983, -0.045008644461631775, -0.5633498430252075, -0.4056597948074341, -0.17718903720378876, 1.1389483213424683, 1.4073301553726196, 0.46625038981437683, 0.17777760326862335, -0.7416609525680542, 0.9140459895133972, -0.5620803833007812, 0.07321354746818542, 0.7490499019622803, -0.5689310431480408, 0.8644686341285706, 0.6213691234588623, 0.38813695311546326, 0.6906439661979675, -0.15174216032028198, 0.5476974844932556, -0.5393266677856445, 2.2801778316497803, 1.0786256790161133, -0.3365044593811035, 0.3015473783016205, -1.4294527769088745, 0.8630412817001343, -1.5183782577514648, 1.1557773351669312, -0.2761836349964142, 0.13596922159194946, 0.29351767897605896, -0.25469473004341125, -0.050514236092567444, 0.20727108418941498, -0.3129894435405731, 0.5134942531585693, -0.02470874786376953, 0.47740480303764343, 1.243359923362732, 0.6876233220100403, 0.8363791108131409, 0.3961212933063507, -1.3232296705245972, 0.12490662932395935, 0.9601711630821228, -0.7152007222175598, -0.5097843408584595, -1.1034237146377563, 0.983727753162384, 0.673211395740509, 0.1256844401359558, -0.833860456943512, 0.558692455291748, 0.2794424593448639, 0.40488266944885254, 1.3281618356704712, -0.2055538296699524, -0.8577921986579895, 0.6314336061477661, 0.09789128601551056, 0.2208210825920105, -0.8639317154884338, 0.01965269446372986, 0.3789984881877899, 0.9819303154945374, -0.3970579504966736, -1.3423810005187988, 0.32324856519699097, -0.04873540252447128, 0.6514636278152466, -0.7363355159759521, -0.0030362228862941265, -0.2723965644836426, -0.5520656704902649, -0.3504524230957031, 0.4232366979122162, -0.316232293844223, 0.7353672385215759, 0.5759274959564209, -0.8014507293701172, 0.30317384004592896, 2.3202755451202393, 0.5876634120941162, -1.725929856300354], [0.8026155829429626, 0.7102307677268982, 0.6451478004455566, -0.269612193107605, 0.6163864731788635, 0.355998158454895, -2.6578426361083984, -1.0780940055847168, -0.3769061267375946, 2.138927459716797, 0.8173722624778748, -1.4889287948608398, 1.385215163230896, -0.14323526620864868, 1.6279845237731934, -0.9976480007171631, -0.040858589112758636, 1.9347950220108032, -0.17653918266296387, 0.6461256146430969, -0.5010639429092407, 0.911249577999115, 1.8683793544769287, 0.9918854236602783, 0.6623285412788391, 1.804895281791687, 0.8327739238739014, 1.9159841537475586, -0.26195982098579407, 0.37040916085243225, -0.35857924818992615, -0.2611192464828491, 1.681963562965393, -0.7159780859947205, -0.3880980610847473, 0.5034023523330688, -1.2792564630508423, 1.6590195894241333, 0.3516051173210144, -0.537039041519165, 0.7093722820281982, 1.137725830078125, 1.353785514831543, -0.3038044571876526, -1.1314080953598022, 0.24301457405090332, 1.0896245241165161, 0.5400462746620178, 1.2623318433761597, -0.38871604204177856, 0.5878885984420776, 1.2781071662902832, -1.2853020429611206, -1.229544997215271, -0.37801486253738403, -0.095171257853508, -1.680314302444458, -1.385546326637268, -0.4853668808937073, -0.2759729325771332, -0.6753403544425964, -0.6249889135360718, 0.7792966365814209, 2.024217367172241, -0.6806603074073792, -0.10884439945220947, -1.6084849834442139, 1.4775526523590088, 1.0206609964370728, 1.3510651588439941, 2.4148964881896973, -0.5493259429931641, -1.0984874963760376, -0.1958955079317093, -0.16515061259269714, -1.1688604354858398, 1.3008403778076172, -1.9475098848342896, -1.3142071962356567, -0.7805539965629578, 1.1081650257110596, 0.4296443462371826, 0.20396867394447327, 0.8359545469284058, 0.1552858203649521, 0.07126142829656601, 0.8623380661010742, 1.8630807399749756, 0.45474672317504883, 0.3995457887649536, -0.3611268103122711, -0.15570613741874695, 0.14647255837917328, -0.10410735011100769, -0.5608302354812622, -0.999806821346283, -0.14178423583507538, 0.2820449471473694, 1.1025285720825195, 2.820695400238037, 0.4976065456867218, -0.4603511393070221, 0.24798700213432312, -0.7869279384613037, 1.2128130197525024, 0.9595357179641724, -0.45064765214920044, 1.0494542121887207, 0.5033331513404846, 2.0010716915130615, -0.9481068253517151, -1.666926622390747, -1.7203036546707153, -0.5586159825325012, 1.010411262512207, 1.3870457410812378, 1.3481794595718384, -0.028919987380504608, -0.36733511090278625, -0.03217808157205582, 0.1995086967945099, 1.0614835023880005, -0.522196888923645, -0.6583558320999146, 0.0671318843960762, -0.5341055989265442, 1.3377082347869873, -1.7413909435272217], [0.21310612559318542, 0.24511957168579102, 1.6602405309677124, -0.23770171403884888, 0.2953520119190216, 0.30285996198654175, 0.007288586813956499, -0.25855275988578796, -2.235724687576294, -1.2658138275146484, 0.6045657396316528, -1.692233681678772, -0.7897747159004211, 0.7223923206329346, -0.890565812587738, 0.5870625376701355, 1.9768625497817993, 1.182778239250183, -0.23985205590724945, 0.0364183746278286, 0.8942501544952393, -0.4716412425041199, 0.8127124905586243, 0.04698106646537781, 1.0585051774978638, 0.30137279629707336, -0.32336482405662537, -0.4503672122955322, -1.2936137914657593, 0.563993513584137, -0.2870418131351471, -0.7125111818313599, 0.7234960794448853, -0.012375996448099613, 0.11932238936424255, -0.044548485428094864, 0.6158579587936401, 0.3907325565814972, -1.0968050956726074, -0.42137864232063293, 0.03200346231460571, 0.05001510679721832, -2.042642593383789, 0.6204112768173218, -1.2135919332504272, -0.6517060399055481, 0.42395442724227905, -0.5133547186851501, 1.3606995344161987, 0.3809228539466858, -0.9497675895690918, 0.8430781960487366, 0.4640395939350128, -2.174170732498169, 0.7721874713897705, 0.2952338755130768, 0.22879596054553986, 0.4387497305870056, -0.1255018562078476, -2.2828094959259033, 0.5901933312416077, 1.1048871278762817, -0.5769066214561462, 1.1111774444580078, -0.46426230669021606, -0.9557783007621765, 1.042396068572998, -1.3467249870300293, 0.8819721937179565, -1.2602965831756592, -2.087491273880005, 1.3593240976333618, -0.32991132140159607, 0.8121670484542847, 1.729248285293579, 0.3591395914554596, 1.34725821018219, -0.19903966784477234, 0.29238462448120117, 2.301600217819214, 0.25358110666275024, 1.1313505172729492, -0.8588704466819763, -0.8368743062019348, 0.9337127804756165, 0.34660404920578003, 0.7444605827331543, -2.07725191116333, 1.5785155296325684, -0.424725741147995, -1.1830655336380005, -0.49338820576667786, 0.316983163356781, -0.9872938394546509, 0.31020480394363403, -0.05329159274697304, 0.4692935347557068, 2.359999895095825, 1.0440946817398071, -2.2010467052459717, -1.0425511598587036, 0.3925487995147705, 0.6616230607032776, 2.5925819873809814, -0.23777036368846893, 1.6610931158065796, -0.3243774473667145, 2.4044182300567627, -1.3243662118911743, 0.7871351838111877, 1.367166519165039, 0.8384739756584167, 0.512917160987854, 0.32124316692352295, 0.9523928165435791, 0.7879519462585449, 0.8989421129226685, 0.30290165543556213, -0.17772932350635529, -0.9071411490440369, -0.14820168912410736, 0.10836759954690933, 0.12474188953638077, -1.4332051277160645, -2.295827627182007, -1.3624955415725708, -0.1644139140844345, -1.1522271633148193], [-1.637986660003662, 0.738125741481781, -0.6774051785469055, 0.4592491388320923, -0.3715366721153259, -1.0962026119232178, 0.08246994018554688, 1.1769804954528809, 0.2436036765575409, -0.03461221233010292, -1.5961546897888184, 0.6211991310119629, 2.035123109817505, -0.3185581564903259, 0.5234798789024353, -1.4491125345230103, 0.6221354603767395, -2.742198944091797, 0.04780952259898186, 0.37679779529571533, -0.3518877923488617, 0.6941870450973511, 1.8256466388702393, 1.0720049142837524, -0.3690791130065918, 0.39026275277137756, -0.4770588278770447, 1.2869775295257568, 1.2869683504104614, 0.05974571034312248, -0.6363973021507263, -1.0147292613983154, 0.702061653137207, 0.7473598122596741, 1.8378326892852783, 1.713668942451477, -1.1267898082733154, 2.3497142791748047, 0.8091829419136047, 0.9588602185249329, 0.8245735764503479, 1.2262190580368042, 0.5595541596412659, -0.27721214294433594, 0.05685257911682129, -0.4079541563987732, 0.6447554230690002, -1.8132691383361816, -0.9985966086387634, -2.871609926223755, 1.5459057092666626, -0.15273001790046692, 0.45487335324287415, 0.837251603603363, -0.8185442686080933, 0.3074153959751129, 0.2352675199508667, -1.0358189344406128, -0.35353466868400574, 1.8720364570617676, 0.580535888671875, 0.4474434554576874, 0.4971832036972046, 0.4662570059299469, -0.5451416969299316, 0.14628763496875763, -0.4629269242286682, -1.033962607383728, 0.2873956561088562, -0.7335017919540405, 0.09637454152107239, -1.6332321166992188, -0.7917255759239197, -0.03842857480049133, -1.7739566564559937, -0.28692352771759033, -2.0166404247283936, 0.6727204918861389, 1.2765593528747559, -1.2189953327178955, 1.1449352502822876, -0.5360578894615173, -0.03919437155127525, 1.090104579925537, 0.2882288992404938, 1.315140962600708, -0.6365870833396912, 0.3516559302806854, -1.634277582168579, 2.1112542152404785, -0.6812499761581421, 0.873129665851593, -0.39998510479927063, 1.4079633951187134, 0.346800297498703, 0.011656287126243114, 0.6849363446235657, 2.1284682750701904, 0.40657860040664673, 1.129550576210022, -0.6562879085540771, 1.3868281841278076, -0.5671058893203735, 0.10433199256658554, -1.479398488998413, 0.3630703091621399, 0.731853723526001, -0.12405253201723099, 0.2770279347896576, 0.6598244905471802, 0.4747857451438904, -1.1947892904281616, -1.3667960166931152, -1.076537847518921, 1.5180399417877197, 0.23426954448223114, 0.05922738090157509, 0.1834670901298523, -0.9432873129844666, -0.08740843087434769, -0.8507233262062073, -0.20668593049049377, -0.3334406316280365, -0.9618349075317383, 0.007496733218431473, 0.4781697690486908, 0.8171724081039429, -0.0926203653216362], [0.7064865231513977, 1.1764756441116333, 0.3537314236164093, 0.2945033609867096, 1.3205004930496216, -1.9507100582122803, 1.6884530782699585, -1.734784483909607, -1.0919626951217651, -0.9746865630149841, -1.0086438655853271, -0.2280411422252655, -0.4339368939399719, 0.9002059698104858, -1.3724842071533203, -1.1974183320999146, -2.391010284423828, -0.1164950504899025, -0.5215909481048584, 0.03781208395957947, -0.15763629972934723, -0.10481777042150497, 0.952241837978363, -0.6941289901733398, 0.7857941389083862, 0.2889988422393799, 0.17399469017982483, 0.6260025501251221, 0.30950987339019775, -0.900405764579773, 0.13683630526065826, -0.09284657984972, 0.33514031767845154, -0.3249637484550476, -0.31949758529663086, -1.782650113105774, 0.2392614185810089, -0.7064831852912903, 0.42929011583328247, -0.31160423159599304, -0.7947705984115601, -0.06938625872135162, -0.5054537057876587, -0.7290782928466797, 0.3558950424194336, 0.7365221381187439, 0.7838229537010193, 0.3749607503414154, 1.4320425987243652, 0.13481035828590393, -0.2671639621257782, 0.21590054035186768, 0.4686652421951294, -0.7028423547744751, 2.4716596603393555, 0.10683033615350723, -0.3539391756057739, 0.13636063039302826, -1.234518051147461, 1.0394090414047241, -0.12627731263637543, -0.37487301230430603, 1.0532634258270264, -1.218429446220398, 0.08735440671443939, -0.846866250038147, 1.1122792959213257, 0.6071751713752747, -0.13806715607643127, 1.350571632385254, -0.02058032900094986, 0.7959348559379578, -0.6865409016609192, -2.0945351123809814, 1.4157944917678833, -1.316514253616333, -0.3437105119228363, 0.5340897440910339, -0.9835451245307922, 0.84761643409729, 0.39226803183555603, -0.27058324217796326, 0.27223873138427734, 0.5495288968086243, -0.5580521821975708, 0.8390668034553528, 1.0683218240737915, -2.1265902519226074, 0.7022692561149597, 1.0034253597259521, -0.7301374673843384, 2.0924723148345947, -0.43340492248535156, 0.017647240310907364, -1.3277709484100342, -0.7931362986564636, 1.5401585102081299, 0.3363020122051239, -2.2612457275390625, -0.01235573086887598, -1.2372041940689087, 0.30495765805244446, 1.2220830917358398, -0.19160102307796478, 0.9895550012588501, -0.4660274088382721, 1.7503745555877686, -0.7617714405059814, -1.27378249168396, -0.7248087525367737, -0.21766091883182526, 0.23430071771144867, -0.316098690032959, -1.7536553144454956, -1.296625018119812, 1.0485634803771973, -1.1091418266296387, 0.34086090326309204, -1.799277901649475, -0.7386525273323059, -0.5467721223831177, 1.5807496309280396, -0.738172709941864, -0.4309655725955963, -0.9183440804481506, -0.968413233757019, 0.3042466342449188, 0.23188750445842743], [0.6629714965820312, -1.5424761772155762, -0.17024490237236023, 2.0592143535614014, -0.5771197080612183, -1.1660364866256714, -0.4682371914386749, 0.8070078492164612, -1.0154246091842651, -0.6199190020561218, -0.735388457775116, -0.932923436164856, -0.6107611060142517, 0.2541278004646301, 0.7637208104133606, 0.8823776841163635, 0.15190647542476654, 0.9337490200996399, 0.3895408511161804, 0.8703030943870544, 0.5602406859397888, -0.3701317608356476, 0.24840770661830902, 2.0958776473999023, 0.22391092777252197, -1.3780310153961182, -0.5044471621513367, -0.47759711742401123, -0.45187652111053467, 0.10066177695989609, -1.2192951440811157, -1.0200223922729492, -0.25009194016456604, 0.12294970452785492, -0.22526875138282776, -0.2695176303386688, 1.2231812477111816, -1.884125828742981, 1.0999096632003784, -0.8983811736106873, -0.23992718756198883, -1.163885474205017, -1.224617600440979, 0.5912274718284607, -1.4647365808486938, -0.13315241038799286, 0.07454496622085571, -0.32034510374069214, -0.8631030917167664, 1.597350835800171, -1.3768703937530518, 0.2708432674407959, 1.4748961925506592, -0.03873905539512634, 0.3451329171657562, 0.03334713354706764, 0.37670478224754333, -0.16873084008693695, 1.0622965097427368, 0.9372450113296509, -0.18829137086868286, -1.2460185289382935, -0.42620187997817993, -0.4964478611946106, 1.2737340927124023, -0.3712119460105896, 0.38559582829475403, -1.6136434078216553, -2.1248724460601807, -0.5554711818695068, 1.0160601139068604, -0.950685441493988, -1.149733066558838, -1.1986230611801147, -0.43617236614227295, -1.5596888065338135, -1.1333281993865967, 1.296168565750122, 1.1757718324661255, -1.510191559791565, -0.6517139077186584, 1.9851200580596924, 1.4348971843719482, -0.5467853546142578, -1.903809666633606, -0.07855584472417831, 0.6156954169273376, -0.37528708577156067, -0.9413949251174927, 0.08579231053590775, 0.5739161372184753, -0.7159076929092407, -1.831561803817749, 0.9302056431770325, -1.0694447755813599, 0.21641606092453003, 1.6313787698745728, -0.7498616576194763, 0.23025134205818176, 0.6203989386558533, 1.6944953203201294, 0.7774879932403564, 1.4466913938522339, 0.7881951928138733, -0.7668471932411194, 1.723503589630127, -0.9258549809455872, -0.508813738822937, -0.78703773021698, 0.8967825174331665, -1.1632713079452515, 1.9421840906143188, -1.1151401996612549, 0.07931330800056458, -1.2352073192596436, 0.37089109420776367, 1.3822367191314697, -1.3101418018341064, -0.43100374937057495, 0.7579107284545898, 0.8204591274261475, -0.516977071762085, 2.3267264366149902, 2.7863290309906006, 0.9865851402282715, -1.5139390230178833, -1.0529228448867798, -1.5790307521820068], [-0.11951342225074768, -0.8754047751426697, -1.4830164909362793, 1.2360033988952637, -0.15832526981830597, -0.21300087869167328, -1.071480631828308, 0.09801142662763596, -2.0349414348602295, 0.09192343056201935, 0.3036113381385803, -0.17364582419395447, -1.2824491262435913, -0.7968083024024963, 1.3594305515289307, -0.4044622480869293, -2.027223825454712, -1.571723222732544, -1.6780426502227783, 0.6395906209945679, 0.15092021226882935, -0.9443530440330505, 0.6518012285232544, 0.5479869246482849, 0.03807566314935684, 0.5235965847969055, 0.4209192991256714, -0.6162900328636169, -1.893445372581482, -1.8561335802078247, 1.131379246711731, -0.03033686801791191, 0.3484955132007599, 0.7247894406318665, -0.8099991083145142, -0.7220434546470642, -0.780087411403656, -1.0990194082260132, -0.30610233545303345, -0.21582089364528656, -1.0050076246261597, -0.4452783465385437, -0.20498931407928467, 0.35028883814811707, -0.7916945219039917, 1.2447919845581055, -1.2983119487762451, -0.7362827658653259, 0.1160702258348465, -0.09538530558347702, -0.6209594011306763, 0.7601005434989929, -0.5436416864395142, -0.5037354826927185, 0.5740318894386292, 1.3170850276947021, 0.5798243284225464, 0.07987494021654129, 2.160609006881714, 1.0271937847137451, -0.7917320728302002, -0.4403468668460846, 0.36302322149276733, -0.2712448835372925, -0.1565679907798767, -0.9252164363861084, 0.2351434975862503, -0.022375892847776413, 0.4592900574207306, -0.8292474150657654, -1.1304765939712524, 0.4857981503009796, -2.0295827388763428, -1.7803024053573608, -2.7162857055664062, -0.6069526672363281, -0.0458226352930069, 0.29511651396751404, 0.023483678698539734, -0.6567687392234802, 1.3615261316299438, 0.5242623090744019, -0.2597314715385437, -2.1647582054138184, 0.09626226872205734, 0.36305466294288635, 1.4626009464263916, -1.967166781425476, -0.9150981903076172, 2.055642604827881, 0.5384026765823364, 0.31208470463752747, 0.2991649806499481, 0.6484252214431763, 0.31390732526779175, 0.6542214155197144, 0.5608898401260376, 1.380125641822815, 0.32202380895614624, 1.1579868793487549, 0.16552211344242096, 0.2801377475261688, -0.7870245575904846, -0.7618148922920227, 0.8981662392616272, 0.508488655090332, -0.7833918333053589, -0.8912780284881592, 0.24862739443778992, 1.0047610998153687, -0.043935392051935196, -0.7179407477378845, -0.07507377862930298, 1.3480629920959473, 0.021895363926887512, 1.6636019945144653, 0.057208720594644547, 0.23768959939479828, -3.0295116901397705, -0.05082975700497627, -0.3462686240673065, 0.1823350042104721, 2.962412118911743, -0.1778475046157837, 1.3807145357131958, 0.3658243715763092, -0.45783916115760803, 0.11679752916097641], [0.7511091232299805, 0.6562480330467224, 0.23697905242443085, 0.054506704211235046, -0.19563919305801392, -1.1626365184783936, -2.333488702774048, -1.5509557723999023, 0.1547730267047882, -1.3555026054382324, 0.7653663754463196, -1.24690842628479, -0.23847299814224243, 0.06156817078590393, -0.5049651265144348, -0.4857489764690399, -0.7598157525062561, 0.3089573085308075, 0.5404300093650818, -0.36695346236228943, -0.3575601577758789, -0.6350598931312561, 0.4330834746360779, 0.49065297842025757, -0.29058119654655457, 0.19403746724128723, -0.645831286907196, -1.3666006326675415, -0.43571460247039795, 0.3359711170196533, -0.9633684158325195, 0.929318904876709, 0.6756474375724792, 0.9221687912940979, -1.6341228485107422, 0.5978209376335144, 0.14616303145885468, 1.016175627708435, -0.20702669024467468, -1.1790692806243896, -0.5980297327041626, 2.246912956237793, -1.6285452842712402, 0.8344954252243042, -1.3393566608428955, -1.032881498336792, -0.14726144075393677, 1.4854646921157837, 1.1522501707077026, -1.5658764839172363, -0.2595078647136688, -0.7658659815788269, -0.07234556972980499, -1.7249399423599243, -1.6248759031295776, -0.5035088062286377, -0.26735374331474304, 0.7259225249290466, -0.29520148038864136, 0.4789138436317444, 0.33459293842315674, -2.304093360900879, 0.03632044419646263, 0.6219327449798584, 1.5409218072891235, 2.1337339878082275, -0.4886571168899536, 0.4195937216281891, 0.7974591851234436, -0.3915258049964905, -1.0457485914230347, -1.0758159160614014, 0.7327509522438049, -0.3538793623447418, 2.176435708999634, -0.07312063872814178, -0.5344631671905518, 0.7901189923286438, 0.492138147354126, -1.1769300699234009, 0.7666853666305542, -0.01953931897878647, 0.850861132144928, 0.23458901047706604, -0.31055891513824463, -0.3125625550746918, -0.2695866823196411, 0.6480415463447571, -0.6031395196914673, -0.5270475149154663, 0.12856678664684296, 0.10873623937368393, -0.9360108971595764, 1.325186014175415, 0.6785508990287781, -0.090887151658535, 0.19448311626911163, -0.24381764233112335, -1.0404468774795532, 0.1658141165971756, 1.9138057231903076, -0.4932156205177307, -0.6204642057418823, -1.0881277322769165, -0.0254130307585001, -0.6384538412094116, -1.7381329536437988, -0.01651846431195736, 0.43791165947914124, -1.8951612710952759, 1.1716725826263428, 0.6854058504104614, 1.2860674858093262, -0.8535226583480835, -0.03137429431080818, -1.293586015701294, -2.3908679485321045, -0.8682246804237366, 0.9163094162940979, -0.1676916480064392, -0.15833376348018646, -0.16849367320537567, 0.026545917615294456, -2.9425666332244873, 0.5738319158554077, 0.5928924679756165, 0.1470668762922287, 1.8801909685134888], [1.566630244255066, -1.297906517982483, 1.4577873945236206, -1.4316293001174927, 0.5374217629432678, 0.14930656552314758, -0.9806929230690002, 0.15055806934833527, 0.08896303921937943, -0.034890323877334595, -0.4617496430873871, -0.42945677042007446, -1.1183359622955322, -1.4646904468536377, -0.9776597619056702, 0.685247004032135, 1.5979912281036377, -0.01632213406264782, -0.19246378540992737, 1.6505597829818726, 0.1339527666568756, 0.5087787508964539, 0.547678530216217, -0.03618606925010681, -1.5267257690429688, 0.6819407343864441, -0.2237502932548523, 0.6866661310195923, -1.1461818218231201, -0.40062981843948364, -0.5004865527153015, -1.2629563808441162, 1.2989870309829712, -0.6383994817733765, -0.07566779106855392, -0.13680757582187653, -0.10383359342813492, -1.6839834451675415, 1.55570650100708, -0.6361481547355652, 2.17596697807312, 0.6152699589729309, -0.2009383887052536, 0.19388842582702637, 0.5828717947006226, -0.29556921124458313, -0.3997981548309326, -0.3251916468143463, 1.2966177463531494, 0.009717727079987526, -0.7098320126533508, -0.15630804002285004, -0.15988405048847198, -0.6457483172416687, 0.0735374391078949, -0.1540084034204483, 0.6800790429115295, -0.11363749951124191, -0.13192927837371826, -0.22964206337928772, -0.4843633770942688, -1.1384470462799072, -0.9326533079147339, 0.8371725678443909, 1.023270606994629, 0.8257899880409241, -0.8537322878837585, -0.6907775402069092, -0.7939041256904602, -2.1871161460876465, -1.0175701379776, 1.2337287664413452, 0.7068626880645752, -1.2514859437942505, 1.1116951704025269, -2.0904390811920166, -0.4526495337486267, 1.245459794998169, 2.1326401233673096, 1.3565635681152344, 0.3689878284931183, 0.2739306390285492, 0.7120552062988281, -2.2147085666656494, 0.5785764455795288, 1.5318363904953003, 0.42534127831459045, -1.80707848072052, 1.9844629764556885, -0.651370108127594, -0.13704587519168854, -0.9695618748664856, 0.8575971126556396, -0.5580468773841858, -0.7933425903320312, -0.3357572555541992, -1.3106303215026855, 0.11568030714988708, 0.7498899698257446, 0.5989921689033508, -1.7561204433441162, 1.8465358018875122, -0.4873165488243103, -0.3384323716163635, -0.816257894039154, 0.8022164106369019, 0.9386136531829834, 1.1867469549179077, 2.3914544582366943, -0.059088412672281265, -0.5758297443389893, 0.240123450756073, -0.5313227772712708, 0.5910547971725464, -0.29562610387802124, 0.2356543391942978, 0.6217742562294006, 0.7967365384101868, -1.305288553237915, -0.06930964440107346, 0.45226699113845825, 0.5641772150993347, -0.1791602373123169, -1.2556912899017334, 1.6848779916763306, -0.7854236364364624, 1.3339415788650513, -0.332387238740921], [-0.5472126603126526, -0.6983315944671631, -1.5392217636108398, 0.07124804705381393, -0.7286600470542908, -1.0762290954589844, 0.09096631407737732, 0.01915290765464306, -0.41958683729171753, 0.6202212572097778, 0.442914754152298, 1.3464107513427734, -1.0293171405792236, 0.31198132038116455, -0.7942719459533691, -0.06465616822242737, -1.5940877199172974, -0.946142315864563, -0.10591944307088852, -0.6582507491111755, -0.4795167148113251, -0.9031358361244202, 0.3077089190483093, 0.36855143308639526, 0.25035178661346436, 1.111741065979004, 0.49708032608032227, -0.2864929437637329, -0.6881508827209473, 0.009881916455924511, 0.5077013373374939, -0.9015211462974548, -0.5019708871841431, 0.6324177980422974, 0.3656822144985199, -0.8932254314422607, -1.9109605550765991, 1.4296237230300903, 0.09394074231386185, -0.0075492849573493, 0.018761973828077316, 0.6109524965286255, -0.2122918665409088, 0.5243521928787231, -0.1100437194108963, 0.5296132564544678, -0.19581249356269836, 0.14321789145469666, -0.07898853719234467, 0.30893296003341675, 0.31700652837753296, 0.9608911871910095, 2.3718247413635254, 0.457732617855072, -0.38034787774086, 1.0663632154464722, 0.1487794667482376, -1.489660620689392, 0.9713208675384521, -0.05396192893385887, 1.0404672622680664, 0.02400762401521206, -0.05835878849029541, -0.06054497882723808, 1.2963074445724487, -0.5679718255996704, 1.1850022077560425, -0.05263308435678482, -0.718139111995697, -0.0249774307012558, -0.16590824723243713, -1.7132595777511597, 1.5905495882034302, 0.36181363463401794, 1.1110866069793701, 1.8089916706085205, -0.3279261589050293, -0.6856260299682617, 0.4429871141910553, -0.080536849796772, -0.5614732503890991, -1.0067919492721558, -1.0489505529403687, 0.7566754221916199, -0.6550727486610413, -0.4145353138446808, 0.7912958264350891, -0.2179972380399704, -0.36003753542900085, -0.039745528250932693, 0.32183393836021423, -0.22378505766391754, 1.4686518907546997, 1.305302619934082, -0.3947727382183075, -1.6706573963165283, 1.0079575777053833, -1.2871463298797607, -0.17269086837768555, 1.9544751644134521, 0.41884511709213257, -0.6173960566520691, 1.7438346147537231, -0.2875271737575531, 0.46043166518211365, -0.3414105176925659, -0.6930782794952393, -0.6154007911682129, -0.6648003458976746, 0.8715904951095581, 1.115127444267273, -0.9762346744537354, -0.1760515719652176, 0.5389069318771362, 0.23154792189598083, -0.7121157050132751, -1.8045804500579834, -0.9497490525245667, 1.0254497528076172, 0.8912012577056885, 0.40713584423065186, 1.3786249160766602, -0.9270107746124268, -1.4719634056091309, -0.6882910132408142, 0.7473226189613342, 1.3379905223846436, 0.4362519383430481], [0.47125548124313354, -0.6189939379692078, 2.0814685821533203, -0.2427588552236557, -1.1514713764190674, -0.9611908793449402, 0.16262061893939972, -1.1822035312652588, 1.1746840476989746, 0.2205984890460968, -0.26792457699775696, -1.037065863609314, -2.897704601287842, -0.5774268507957458, -1.2739428281784058, 0.5290217399597168, 1.608672022819519, -1.8442389965057373, -0.005739486776292324, -0.08977053314447403, 0.14235633611679077, -0.8220767974853516, 0.5131890773773193, 0.7856281995773315, -0.6980470418930054, 1.5645359754562378, 0.4324565529823303, 0.2310071885585785, 0.37805262207984924, -0.9704540967941284, -1.8683823347091675, 1.5594046115875244, -0.10178915411233902, 0.01446966640651226, 0.565327525138855, 1.0188922882080078, 1.8734115362167358, -0.7584203481674194, -0.09532205760478973, -1.2948514223098755, 1.5012414455413818, -0.4294155240058899, -0.6991837620735168, -0.9005422592163086, 0.136734277009964, 0.0014120594132691622, -0.09737240523099899, 0.660575270652771, -0.43768396973609924, -0.2553723454475403, 2.390395402908325, 0.4753360450267792, 0.06234269216656685, 0.18686753511428833, 1.0193201303482056, -1.0564463138580322, -0.9968406558036804, -1.1613022089004517, -0.2921708822250366, 0.3422582149505615, 0.17658288776874542, 0.919873058795929, 0.26885971426963806, 0.4193045496940613, 0.9235241413116455, 0.7201040387153625, -0.08032403141260147, -0.39656737446784973, -1.4959526062011719, 1.5691167116165161, 1.2953495979309082, 1.5601764917373657, -1.652463674545288, 0.6598897576332092, -0.2462977021932602, -2.309499740600586, 0.08767368644475937, 0.44422242045402527, 0.8356568217277527, -0.9377472400665283, 0.1365160346031189, 0.7191482186317444, 0.9445116519927979, 0.8958982825279236, -0.25666818022727966, -0.36674943566322327, -0.25505703687667847, -0.44363799691200256, 2.1204302310943604, 0.7467591166496277, -1.7839068174362183, 0.2555396258831024, 0.035803839564323425, 0.3751453161239624, -0.3337458372116089, -0.5228429436683655, -0.760071873664856, -0.9075446128845215, 0.8121622800827026, 1.3310011625289917, -0.11322131007909775, -0.5646083354949951, -1.0093185901641846, 1.1846961975097656, 1.0677101612091064, 0.972001314163208, -0.9995871186256409, -0.04040847346186638, -0.3632190525531769, 1.5021133422851562, -1.4744564294815063, -0.5977907180786133, -0.7581935524940491, 0.704014778137207, 0.8817653656005859, -0.3638586699962616, 0.6995223164558411, 0.10926888883113861, 0.9105433225631714, -1.4953806400299072, -0.4170496165752411, -0.488487184047699, -1.1001758575439453, 2.180877447128296, -0.9430505633354187, -1.6948661804199219, 1.5096259117126465, -0.270683616399765], [1.2647461891174316, 0.7735980153083801, 1.7046412229537964, -0.7389764785766602, -0.8552024364471436, -0.4061911404132843, -0.4925871193408966, 0.7739692330360413, -0.1529235541820526, -1.9719934463500977, -0.7473045587539673, -0.6708125472068787, -0.9897660613059998, -1.2197108268737793, -0.3623439073562622, -0.28516700863838196, 0.6940402984619141, -1.6160356998443604, -0.02103033848106861, 0.002564165974035859, 0.26087918877601624, 0.04037575423717499, 1.391507625579834, 0.16351284086704254, 1.0926947593688965, -0.46824848651885986, 1.1150258779525757, -0.6103026270866394, -0.6180035471916199, -0.23726195096969604, -2.218458890914917, 1.155838131904602, -0.8425688147544861, -0.16307185590267181, 1.1791576147079468, -0.971139669418335, 0.2825625240802765, 1.5120611190795898, 0.5860462784767151, 0.7205662131309509, -0.09754869341850281, -0.45764797925949097, 0.15134398639202118, -2.6057662963867188, 0.33318665623664856, -0.613864541053772, 2.1351072788238525, -1.2650851011276245, 1.383609414100647, -0.12496765702962875, -0.5243707895278931, -1.30001699924469, -0.9686572551727295, 2.53934907913208, -0.7381786108016968, -1.2480638027191162, -0.6191854476928711, 0.9358009099960327, -0.66346675157547, -1.2963476181030273, 0.22084447741508484, -0.34347766637802124, 0.32612892985343933, -0.7722336649894714, -1.0597018003463745, 0.48593056201934814, -0.05877053365111351, -0.12100416421890259, -1.926038384437561, -1.3917640447616577, -0.31052273511886597, -0.057852499186992645, 0.7234990000724792, -1.9202760457992554, 0.06146099790930748, -0.19498705863952637, -1.0893803834915161, -0.6187226176261902, -1.4486628770828247, -0.12926802039146423, 0.16428157687187195, -0.319128155708313, 1.41122567653656, -0.8870476484298706, 1.1353118419647217, -1.335784673690796, 0.7885786294937134, -0.25032249093055725, 0.3742207884788513, 0.13485628366470337, -0.39404264092445374, -1.1890815496444702, -1.3605188131332397, 1.1148959398269653, 0.0003923453623428941, -0.1597810834646225, -0.2151404619216919, 0.5784671306610107, 0.2707521319389343, -0.35544243454933167, -0.13567101955413818, -0.21873213350772858, 1.2093226909637451, -0.21759313344955444, -1.4072209596633911, 2.099675416946411, 0.27490103244781494, -0.6455783843994141, -1.7902865409851074, 1.3338656425476074, -1.1506383419036865, -0.932589590549469, -0.7313740253448486, -0.011574812233448029, 1.1073890924453735, 0.773958146572113, -0.2041294276714325, -1.8600748777389526, -0.3643474578857422, -1.4386272430419922, -0.4453355371952057, -0.1415877491235733, 1.1103425025939941, -1.8244298696517944, -0.9054589867591858, 0.8943250179290771, 0.46324706077575684, 2.243196964263916], [0.6675958633422852, 0.6434422135353088, 0.7129621505737305, 0.624278724193573, -0.3659142851829529, -0.09875573217868805, -0.988084614276886, -0.41630008816719055, 0.004508500453084707, -0.14879584312438965, 1.2928928136825562, 0.7341824173927307, 0.45169955492019653, 0.3873879909515381, -0.6849180459976196, 1.8504676818847656, 1.252914309501648, -0.18688397109508514, 0.23310071229934692, -0.9239957332611084, 0.905327320098877, 2.0364019870758057, 0.7381004691123962, -1.6272695064544678, -1.4329330921173096, -1.1414930820465088, -0.21789155900478363, 1.694307804107666, 0.6385672688484192, -1.5997930765151978, 0.14019794762134552, 0.11434915661811829, -0.052610211074352264, 0.7634206414222717, 0.6515930891036987, 1.0381200313568115, -0.5985394716262817, -0.897225558757782, 0.23258079588413239, 0.2689535617828369, -0.24829138815402985, -0.21526722609996796, 0.9859890937805176, -0.764140248298645, 0.9452970623970032, -0.4438386857509613, -0.22412946820259094, 0.2528385519981384, 0.9848620891571045, -0.45803236961364746, -0.24806874990463257, -1.71905517578125, -0.11982957273721695, -0.4429904520511627, -1.5740246772766113, 0.31579121947288513, 1.8451542854309082, 0.4189421534538269, -1.1478118896484375, -0.6615909934043884, 0.9036681652069092, 0.7016634345054626, 0.9020918011665344, 0.2586890757083893, 0.114423468708992, -1.4460817575454712, 0.3636917769908905, 1.8639779090881348, 1.1111983060836792, -0.22701504826545715, -0.6893501877784729, 1.1083908081054688, 0.9684269428253174, 0.7831174731254578, 0.02568369358778, -0.5883110165596008, 0.5077464580535889, 0.07933726906776428, -1.0242469310760498, 0.13738815486431122, 1.0961757898330688, -0.8989874720573425, -0.03632038086652756, 1.6130746603012085, 0.18464531004428864, -0.3690185844898224, -1.28276789188385, -0.46715399622917175, 0.6235162615776062, 0.31002679467201233, 1.3776637315750122, -0.03552120551466942, 0.41929858922958374, -0.11123060435056686, 1.5175769329071045, -0.09311755746603012, 0.6413828134536743, 1.003013014793396, 2.462190866470337, 1.0085299015045166, -1.137082576751709, 0.29030394554138184, 0.03989189863204956, 1.0797319412231445, -0.5204418301582336, 0.051698196679353714, -0.23449239134788513, 0.2422703206539154, 2.465101718902588, 0.7401367425918579, 0.20470456779003143, 0.3639450669288635, 2.231090545654297, -0.0746520534157753, 1.7640999555587769, -1.682868242263794, 0.5162203311920166, 0.4477718770503998, -1.6188832521438599, -0.616919994354248, 1.0019465684890747, -1.0139882564544678, -0.23831282556056976, 0.014113848097622395, 0.11786109954118729, 0.2515692710876465, 1.4717572927474976, -1.5264171361923218], [-0.055716581642627716, -0.4749114215373993, -0.20273733139038086, 0.4094754755496979, 2.4369451999664307, -0.07333201169967651, -2.900669813156128, -0.10060327500104904, -0.5716846585273743, -0.27091994881629944, -0.17556963860988617, -0.1668553650379181, -0.4183824360370636, -3.060596227645874, 0.9082139134407043, -0.5293890833854675, -0.23187175393104553, -1.678072214126587, -0.10278062522411346, -0.5878168344497681, 0.7974352836608887, -0.08159667253494263, -1.5644607543945312, -2.129565477371216, 0.14495886862277985, 0.2844133973121643, -0.9372637271881104, 1.141406774520874, -0.45866748690605164, -0.3931798040866852, 2.292126178741455, 1.4236671924591064, 1.547296166419983, -0.7184459567070007, -0.9686576724052429, -0.4634368419647217, -0.6115188002586365, -0.9128065705299377, 1.0512899160385132, 0.37161484360694885, 0.3122378885746002, -0.4495386481285095, -0.7513462901115417, -0.5806728005409241, -2.1541991233825684, 0.03491036221385002, -1.235608696937561, 0.17177145183086395, -0.5076690912246704, -0.4373459219932556, 0.5285946130752563, -1.3310210704803467, -1.4828298091888428, 1.0662615299224854, 0.5155418515205383, 0.42878246307373047, -0.6281706690788269, -0.43839383125305176, -1.016052484512329, -0.7667568325996399, -0.4425341784954071, -1.0168918371200562, -0.7451484799385071, 0.3976893723011017, -0.7179211378097534, -0.631462812423706, 0.2137852907180786, -0.28800034523010254, 0.5024836659431458, 0.5398995280265808, -0.20457951724529266, -0.2817997932434082, 1.1209797859191895, 0.00019252393394708633, 0.4449605643749237, 0.4937931001186371, -0.7059947848320007, -0.22395028173923492, 0.13787683844566345, 0.23649106919765472, -0.25315096974372864, 0.3535233736038208, 0.855958104133606, 0.8126344084739685, -0.12591461837291718, 0.5964646935462952, 0.02521231211721897, -1.2611616849899292, -0.22382959723472595, -1.2769461870193481, 0.2804637551307678, -0.6126474738121033, 0.14053106307983398, -2.903749465942383, -2.409885883331299, 0.7545121312141418, 0.23075778782367706, 1.2100379467010498, -1.4957741498947144, 0.656723141670227, -0.7986869215965271, 1.7499964237213135, 1.4631589651107788, 0.037104759365320206, -0.5868706107139587, -0.591196596622467, 0.10313054174184799, 1.0725442171096802, -0.8826273679733276, -1.3136112689971924, -0.40282702445983887, 0.030576875433325768, -1.6237068176269531, -0.619434654712677, -0.7124428749084473, 0.9474385380744934, 0.46550580859184265, 0.28023505210876465, -0.7786675691604614, -0.1327488273382187, -0.568129301071167, 0.8720224499702454, -0.8600283265113831, 0.046773336827754974, 0.28938350081443787, -0.1195683628320694, -1.7123769521713257, 1.4956218004226685], [-0.3693617582321167, -0.46597030758857727, -0.11426199972629547, 0.7758885622024536, -0.20513610541820526, 1.1022616624832153, 2.2727203369140625, -2.002286672592163, -2.2787880897521973, -0.11901319026947021, -0.6164646148681641, 0.2521354556083679, 1.3765928745269775, -0.6600401997566223, 1.1158825159072876, 0.05554789677262306, -1.1667903661727905, 0.702239990234375, 0.7679377794265747, -0.24836421012878418, 1.0340867042541504, 0.5062678456306458, 0.9335306882858276, -1.8008309602737427, -0.22047767043113708, -0.6874139904975891, 1.6063635349273682, -0.30877408385276794, -0.37678512930870056, 1.3137798309326172, 0.037038810551166534, -0.20940355956554413, -1.9579405784606934, 1.5018413066864014, 1.278969407081604, -0.5483583807945251, 1.555220603942871, -1.1379212141036987, 0.288718044757843, -1.3872482776641846, 0.28449156880378723, -0.17510727047920227, 1.603703260421753, -0.9038145542144775, -0.5030839443206787, 0.26812854409217834, 1.0982065200805664, -2.102910280227661, 0.07161147892475128, -1.8401206731796265, -1.4504897594451904, 0.8918229341506958, -1.4091767072677612, -0.5404724478721619, -2.3595621585845947, 0.9325235486030579, 0.7954818606376648, 1.6934486627578735, 0.6984764337539673, 0.6158711314201355, -0.43631505966186523, -0.6259152889251709, -1.1101760864257812, -0.6086127161979675, -0.6465911865234375, 0.31411027908325195, -1.2072795629501343, -0.10190454125404358, 0.3183797001838684, -0.5993025898933411, 0.026102744042873383, 1.3733737468719482, 0.7193411588668823, -0.23497071862220764, -0.3451276421546936, -1.191885232925415, 0.6392600536346436, 1.0715000629425049, 1.47970712184906, -1.6333234310150146, -1.4173510074615479, -0.2012729048728943, -0.6879163384437561, -0.7085995674133301, 0.6785327792167664, -0.29533520340919495, 0.6372120976448059, 1.2204066514968872, -0.19574017822742462, 0.40478503704071045, -0.006760458927601576, -0.6710575819015503, -1.477565050125122, -0.40143200755119324, -0.31026244163513184, -0.37268051505088806, -1.4174176454544067, 0.5903137922286987, 1.1381672620773315, -2.842297077178955, -0.8039877414703369, -0.11007653176784515, -0.3855789601802826, -2.181661605834961, 0.3956278860569, 0.5857276320457458, -1.7344696521759033, 0.9487010836601257, -0.06851475685834885, 1.1851253509521484, 1.1991865634918213, -0.0943945050239563, 0.16276662051677704, -0.5595384240150452, 0.26578736305236816, 1.48464834690094, -1.0383838415145874, -0.29041534662246704, -0.3016277849674225, 0.9759076833724976, -0.0015005021123215556, -0.5400540232658386, 0.35233673453330994, 0.835934579372406, 0.10883768647909164, -0.7081891894340515, 0.9482529759407043, -0.6729205250740051], [-1.233659267425537, -0.1354122906923294, -0.25120168924331665, 1.5830172300338745, 0.12286804616451263, -1.3996050357818604, 0.2794136106967926, -0.17604507505893707, 1.0417916774749756, -1.277873158454895, -0.5008133053779602, -1.4141321182250977, -1.8021305799484253, -1.0684740543365479, -1.7465237379074097, -0.5779337882995605, 1.7730867862701416, -0.34858521819114685, -1.59246826171875, -0.13336731493473053, 0.3032667636871338, -0.17246410250663757, 0.28801795840263367, 1.8374028205871582, -0.6468340754508972, -0.846642017364502, -0.5857062339782715, -0.35752731561660767, -0.9020398259162903, 0.6451420187950134, 0.9954351186752319, -1.036617398262024, -0.32136887311935425, -0.9428009986877441, -0.43383902311325073, -0.5467720627784729, -0.7539215683937073, 2.523474931716919, -1.3531800508499146, 1.039101004600525, 1.6361943483352661, -2.4808928966522217, 1.8987339735031128, 1.51237952709198, 0.7710910439491272, 1.0945074558258057, -1.03848135471344, 0.6499107480049133, 0.6679115295410156, -1.523846983909607, 0.8293505907058716, -1.499992847442627, 0.7323863506317139, -0.4907877445220947, 1.5137100219726562, -0.6330452561378479, -0.5122068524360657, -0.1297670155763626, 0.22524571418762207, 0.40412116050720215, 0.9554161429405212, 0.5325267910957336, 0.4438616633415222, 1.0498708486557007, -1.1934922933578491, 0.5624244809150696, 1.2153924703598022, 2.1321632862091064, -0.24178634583950043, 0.8459519743919373, -0.7201826572418213, 1.0767358541488647, 1.304887056350708, 1.6768572330474854, 0.4774029552936554, -1.1327028274536133, 2.01884388923645, -2.218315839767456, 0.7896257042884827, -0.2675558924674988, 0.3791607618331909, 0.3774474859237671, -1.741716980934143, 0.21941153705120087, 1.3254783153533936, -1.2843036651611328, 1.2046104669570923, -0.8683457970619202, 0.5796394348144531, -1.2787978649139404, 0.2033480852842331, -0.7847191095352173, -0.8410245776176453, -0.9454007744789124, -0.8700456023216248, -0.8471996188163757, 1.0380620956420898, 1.9889980554580688, 0.5242739915847778, 1.4456815719604492, -0.5356647372245789, 0.03335629031062126, 0.38641712069511414, -1.8528579473495483, 1.675154447555542, -0.6449437141418457, 0.5744662880897522, 2.304835319519043, 0.38177207112312317, -0.37846362590789795, -2.2180662155151367, 1.5040379762649536, -0.45790085196495056, -1.483780026435852, 2.031986713409424, 0.24655739963054657, -0.6568881869316101, -0.4513814151287079, -0.13346117734909058, 1.1569385528564453, -0.3743320107460022, 0.9336990714073181, -0.2945297062397003, -1.3562383651733398, 0.2740364968776703, 0.6903754472732544, -1.1842600107192993, -0.6734015345573425], [-1.6705681085586548, 1.0597312450408936, -0.3928287923336029, -0.24762003123760223, -0.4332501292228699, 0.7031866312026978, -1.0647773742675781, -0.8824635744094849, -1.0064315795898438, -0.541918158531189, 1.396437168121338, -1.0079641342163086, 1.2617214918136597, 1.0875343084335327, -0.0897863358259201, -1.8144890069961548, -0.5491622686386108, -1.1975817680358887, 0.6063149571418762, 0.11360806226730347, 1.1007484197616577, 0.2376052439212799, 0.7105314135551453, 0.6793480515480042, 1.2513103485107422, -1.7469772100448608, 1.0558077096939087, -0.268811970949173, 0.7585999369621277, -1.7489358186721802, 0.18048939108848572, 0.9387888312339783, 0.28254520893096924, -1.3031022548675537, -0.40693438053131104, -0.38755884766578674, 0.31170526146888733, 0.41684582829475403, 0.04274671524763107, -0.009008887223899364, -1.5118173360824585, 0.32592853903770447, -1.2878813743591309, 0.6564143896102905, 1.174456000328064, -0.48699307441711426, -0.7237759232521057, 1.2200706005096436, -0.5323737859725952, 0.475952684879303, -0.38344135880470276, 0.9172493815422058, 0.3682423532009125, -0.9147320985794067, 0.22676001489162445, 0.7075349688529968, 1.7517949342727661, -0.8158872127532959, 0.1768244355916977, -0.14104381203651428, 0.2616763114929199, 0.10631804913282394, 0.3235204815864563, 0.3593331575393677, -0.38324910402297974, -0.07134567201137543, 1.4858578443527222, -0.7715228199958801, -1.5004069805145264, -0.5874694585800171, -0.22156061232089996, -0.25887688994407654, -0.2988668382167816, 1.1355448961257935, 2.673191785812378, -0.5750311613082886, 1.0889049768447876, -0.605689287185669, -0.3849908113479614, 0.2788323163986206, 0.725056529045105, 0.8506705164909363, 0.9977239966392517, 0.9781882166862488, -0.09515809267759323, 0.8992317318916321, 0.6962037682533264, -1.7198725938796997, -1.3754173517227173, -1.3011293411254883, 0.5118709802627563, 1.3605635166168213, -1.0325700044631958, -0.5149564743041992, -0.08949978649616241, 0.06755596399307251, 0.8249276876449585, -1.8591806888580322, 0.8398029804229736, 1.1585593223571777, 0.8476547598838806, 0.4730587303638458, -0.19305337965488434, -0.5919705629348755, -0.031825434416532516, 0.1911982148885727, 0.6745110154151917, 2.634394645690918, 0.7146672010421753, 0.7731522917747498, -0.9475187063217163, 0.04589022696018219, 0.7422530651092529, -0.6698926687240601, -0.8708621263504028, -2.3987953662872314, 0.8582189083099365, 0.5922999978065491, -0.6366371512413025, -0.8412267565727234, 0.0957416519522667, 0.9499816298484802, -1.5735591650009155, 0.02662605047225952, -0.3453536927700043, -0.8475016951560974, 0.8257227540016174, -0.39778244495391846], [0.5007467865943909, 0.9216036200523376, 0.22322531044483185, -0.5580227375030518, -0.5342227816581726, -1.2247792482376099, -0.951124370098114, -1.2548075914382935, 1.3031378984451294, -0.029295312240719795, 0.09096530824899673, 1.2667045593261719, -0.2820740342140198, 0.4889606833457947, -1.5123074054718018, 1.159019112586975, 0.1041155532002449, -0.11293797194957733, -0.2118954360485077, -0.7253766655921936, 0.15194040536880493, -0.35410043597221375, 0.9745801687240601, -0.18993665277957916, -0.7025418877601624, -0.29229193925857544, -1.4049854278564453, 0.05098703131079674, -0.6662617325782776, 0.9517125487327576, -0.14735101163387299, 0.10572104156017303, -0.7136306762695312, 2.333402633666992, 0.7980008721351624, -0.5291407108306885, -1.8357765674591064, 0.7942498326301575, 0.12801824510097504, -0.628196656703949, 0.8917311429977417, 0.22642002999782562, 0.6813337206840515, 0.556938648223877, 0.5323932766914368, 0.17441724240779877, 0.38336512446403503, -0.21799327433109283, -0.6247856020927429, -0.8342503905296326, -1.1691391468048096, -1.9752434492111206, -0.6580719351768494, -1.2465357780456543, 0.6603354215621948, 1.6185224056243896, -1.2286438941955566, -0.43698862195014954, 1.6955658197402954, 1.2312681674957275, 1.4080395698547363, -1.8355852365493774, -0.860000729560852, -0.752600908279419, 0.46962690353393555, -1.7672595977783203, 1.3155899047851562, -1.5279303789138794, 0.5219779014587402, 1.3312970399856567, -0.724678635597229, 0.8279003500938416, -1.1442803144454956, 0.9183310270309448, 1.9675043821334839, -1.1056934595108032, 0.256352037191391, -0.46541351079940796, 0.7927452325820923, 2.187443256378174, 0.2534622848033905, -0.434234082698822, -0.1223779246211052, -0.3032985329627991, -0.3534841239452362, -0.805182695388794, -0.9335014820098877, 0.20500217378139496, -0.5695949196815491, -0.86721271276474, 1.0391989946365356, 0.6211327910423279, -1.7482635974884033, 1.3487809896469116, -0.20864032208919525, 1.366994857788086, -1.588141918182373, 0.8739800453186035, 0.2359902411699295, -0.9708374738693237, 0.5710773468017578, -0.007881777361035347, -0.15459449589252472, 1.110107421875, 1.8889439105987549, 1.6591566801071167, -0.7613041996955872, 1.1501020193099976, -0.21923820674419403, 0.23415130376815796, 1.4275858402252197, -1.2136493921279907, 0.08652927726507187, 1.3709100484848022, -1.229635238647461, -0.6163012981414795, 0.5012023448944092, 0.4919638931751251, 0.4921776056289673, 0.3272864818572998, -0.4955991804599762, -0.2903292775154114, 0.8823554515838623, -0.1254684031009674, 0.7784620523452759, 2.1741652488708496, -0.1802079975605011, -0.1047133058309555]]], [[[-1.2589480876922607, -0.333595335483551, 0.0433078408241272, 0.5029807686805725, -3.092958688735962, 1.000345230102539, 1.2323379516601562, 0.5455424189567566, -0.3925662338733673, 0.029833631590008736, -0.8043987154960632, -0.5275653600692749, -1.0311115980148315, 1.0652141571044922, 0.6625593900680542, -1.0099374055862427, -0.2117978036403656, -0.7649903893470764, 1.3252818584442139, -0.10240162909030914, 0.04673723503947258, -1.057490348815918, -0.5560937523841858, -0.6835765838623047, 0.43160057067871094, -0.5985526442527771, -1.8270808458328247, 0.5215156078338623, -0.5338792204856873, 0.598850667476654, -0.9517288208007812, -1.1817152500152588, 1.8310071229934692, -1.7781412601470947, 0.15180452167987823, 0.8040915727615356, -0.8260695934295654, -0.6139710545539856, 1.3050848245620728, 0.8171901106834412, -0.04802175238728523, 0.3145899772644043, 0.07883918285369873, 0.18384724855422974, -1.300840973854065, 0.2931510806083679, -0.6961805820465088, -0.2812234163284302, -0.9203717112541199, 0.008755396120250225, -0.05235685408115387, -0.4899666905403137, -1.3914539813995361, -0.7782489061355591, 0.16609540581703186, -1.5272743701934814, -0.9870241284370422, -2.6156766414642334, 1.3817079067230225, 0.5672726035118103, 1.972762942314148, -0.8805143237113953, 0.7806776762008667, 0.8503566980361938, 0.3607763350009918, -1.2565973997116089, 0.8265348076820374, -0.7786139845848083, 0.47379475831985474, 0.5896202921867371, 0.41815540194511414, -0.48280763626098633, 0.6309332251548767, 0.015382209792733192, -0.31295549869537354, 0.4178391098976135, -0.8357042670249939, 0.43869805335998535, -0.15136316418647766, -2.0807371139526367, -0.3950473368167877, -1.1578925848007202, -0.09514809399843216, -0.588850200176239, -1.083373785018921, 0.5196468234062195, 0.8343771696090698, 1.0807819366455078, -1.007912516593933, -0.8052242398262024, -1.5238901376724243, -0.8246645927429199, 0.40916940569877625, 1.0822463035583496, 0.7634907364845276, -0.17089369893074036, -1.4417119026184082, 1.9985568523406982, -0.8297442197799683, -1.2567955255508423, 1.0116972923278809, -0.18524782359600067, 0.8117108345031738, -1.3263410329818726, -0.7339604496955872, 0.052247192710638046, 1.568963885307312, -1.180983304977417, -1.688767433166504, 0.9137154221534729, 0.5406140685081482, 0.8797073364257812, -0.10668456554412842, -0.19322305917739868, -1.552642822265625, -0.38780882954597473, 0.4041930139064789, -0.810660719871521, 0.7966102361679077, 0.35141029953956604, -0.3917590379714966, 0.48283350467681885, 0.07776182889938354, 0.5710077881813049, 0.7883771657943726, 0.4791220724582672, -0.6152142882347107, 0.2960992455482483], [-0.03910248354077339, -0.21234402060508728, 0.4604121148586273, 1.4811902046203613, 0.08596886694431305, -0.7194544076919556, 0.734809935092926, 0.3194722533226013, -0.13073016703128815, -0.5125994086265564, 0.6367749571800232, 1.2080628871917725, -0.74541836977005, 0.6589915156364441, 0.27318042516708374, -1.4916670322418213, 1.301628828048706, 0.8918095827102661, -1.1798121929168701, 0.43887150287628174, 0.3568715751171112, 0.9109583497047424, 1.7733298540115356, 0.5393761992454529, 0.7009121179580688, -0.09871482104063034, -1.3628066778182983, -0.14163292944431305, 0.3708605468273163, 0.2726007401943207, 0.7792450785636902, -0.5402709245681763, -1.0707710981369019, 0.8757749199867249, 1.0223366022109985, -0.3329051733016968, 0.10253246873617172, 0.010003340430557728, 0.3387434184551239, -1.4072093963623047, -0.6217184662818909, -1.6010370254516602, 0.5310805439949036, -0.6953514218330383, 0.2202187180519104, 0.22859366238117218, -0.9747300148010254, 0.22482112050056458, -0.2384144365787506, 0.589943528175354, -0.7602766752243042, 0.6424435973167419, 0.385090708732605, 0.2685520648956299, 0.7642807960510254, 0.9848360419273376, 0.3847109377384186, -0.19144511222839355, -0.46614503860473633, 1.8515334129333496, -0.39058926701545715, 0.18104547262191772, 0.3572697639465332, 0.4270821511745453, -0.543958842754364, 0.04572588950395584, -0.09837038069963455, 0.7370061874389648, 0.09517870098352432, 0.7801414728164673, -0.48486799001693726, -0.1642424613237381, 0.15354298055171967, 0.7432776093482971, 1.5844175815582275, 0.22291123867034912, -0.7920088171958923, 1.608475923538208, -1.1013333797454834, 2.803006649017334, -1.307973861694336, 0.6191278100013733, -1.1532888412475586, 0.5153228044509888, 0.4569258391857147, -0.12469421327114105, 1.1555067300796509, 0.8026435375213623, -0.3762166202068329, 1.2438926696777344, -0.07374967634677887, -0.5632079243659973, 0.5686811208724976, -0.2563685476779938, -0.4478386640548706, 1.2098371982574463, 0.5461686253547668, 0.5352182388305664, 0.09851513803005219, -1.2015305757522583, -0.20402677357196808, 0.4424302279949188, -1.1777563095092773, 0.13588018715381622, -1.7320054769515991, 1.4811698198318481, 2.417989492416382, 0.814992368221283, 0.4435266852378845, 0.11506394296884537, -0.35480374097824097, 1.2760887145996094, -0.35198473930358887, 1.1231136322021484, 0.3866141438484192, 0.9246235489845276, -0.900502622127533, -1.15192711353302, 0.6598271131515503, 0.09849180281162262, -1.4083460569381714, -0.9779816269874573, 0.12208129465579987, -0.9126855134963989, 1.0687061548233032, 0.17374713718891144, -0.403881698846817, 1.2228583097457886], [0.07465840131044388, -0.18400907516479492, 1.0548959970474243, 0.9491714835166931, 0.17803741991519928, 0.6280858516693115, -0.08240988105535507, -0.00022116197214927524, -0.48978734016418457, -1.3633369207382202, 0.1031574159860611, 1.5574359893798828, 0.5510792136192322, 0.2528291642665863, -1.3442423343658447, -0.5734999179840088, -1.2972992658615112, 0.2655472755432129, 0.1754433959722519, 1.332319974899292, 0.2873290479183197, 0.11084529012441635, -0.8161910176277161, -1.2104648351669312, -0.4854676425457001, -0.6953175663948059, -1.1269639730453491, 1.4437931776046753, -0.8397328853607178, -0.9740679860115051, 0.423453152179718, -1.8451228141784668, -0.6526002883911133, -0.25456702709198, 0.8700817823410034, -3.6585161685943604, -0.3757256865501404, 0.10140946507453918, 0.44779103994369507, -0.5479230880737305, 1.7670488357543945, -0.81270831823349, 0.3880493640899658, 0.6885210871696472, 2.61816668510437, 0.748733401298523, 1.1726971864700317, -0.46413514018058777, -0.06849156320095062, -0.21761341392993927, -3.000375747680664, 1.2552205324172974, 1.036740779876709, 0.9770087003707886, 0.45604074001312256, -1.7642366886138916, -0.7269617319107056, 1.0188459157943726, -1.888561725616455, 0.22347930073738098, -0.8838891386985779, 1.1890952587127686, -0.6630417108535767, 0.14685536921024323, -0.39034155011177063, -0.565280020236969, 0.755149245262146, -0.15901486575603485, -0.04196203127503395, 0.7160196304321289, -0.1539987474679947, 1.6573290824890137, 0.6189336776733398, 0.1827431619167328, -0.6585545539855957, -0.3729196786880493, 2.403533458709717, 0.9995742440223694, 1.2938282489776611, 0.6250452399253845, -0.488128662109375, 0.12453704327344894, 1.538879156112671, -1.4582709074020386, 0.8774728178977966, -0.32211825251579285, 1.2710940837860107, -1.1827424764633179, 0.059682633727788925, 0.7405533194541931, 0.632567286491394, 0.05423540249466896, -1.0594841241836548, -0.1013757735490799, -1.166748285293579, -0.12636937201023102, -0.44000712037086487, 0.7298099994659424, 0.12396208196878433, -1.439427137374878, 0.2012253701686859, -1.2881224155426025, 0.34604382514953613, -0.43500250577926636, -1.6408997774124146, 0.26599353551864624, -1.3111975193023682, 0.1474376916885376, 1.943983793258667, 0.2331923544406891, -0.3941614329814911, 0.6352546811103821, 0.7864210605621338, 0.8320254683494568, 0.7951154708862305, 0.7703448534011841, -0.6964253187179565, -0.04633992165327072, 3.3226993083953857, 0.6615602970123291, 0.5272900462150574, 0.7611525058746338, 1.1515148878097534, 0.08536411076784134, 0.25012120604515076, 0.10878986865282059, 0.8565364480018616, 1.2595628499984741], [0.21344543993473053, -0.49211177229881287, 0.6335144639015198, -1.8557064533233643, 0.16207030415534973, 0.8550379872322083, -0.4869137108325958, -0.8671894669532776, 0.33400866389274597, -0.46367591619491577, 0.9125679731369019, -0.5860824584960938, -2.2520601749420166, 1.2806118726730347, -0.7741931676864624, -1.1679250001907349, 1.1805377006530762, -0.393957257270813, -0.09653020650148392, -0.9913179874420166, 0.12752948701381683, 1.272359848022461, 0.010250350460410118, -0.4918193221092224, 0.354900598526001, 0.5693711042404175, 0.589542806148529, 0.33654695749282837, 0.386165052652359, -0.44639289379119873, -0.3591490089893341, 0.9135409593582153, -0.43904489278793335, -0.4611247777938843, 2.4415957927703857, -2.33028244972229, -0.9020301103591919, 0.5115411281585693, -0.5444105863571167, 1.436766505241394, -1.4424065351486206, 1.1192903518676758, 0.3825104534626007, -1.5552842617034912, -1.4087046384811401, -0.38895857334136963, -2.436840057373047, -0.40806111693382263, -1.716662883758545, -0.6013790965080261, -0.10829780250787735, -0.007535883225500584, 0.7685093283653259, 0.8214373588562012, 0.06418630480766296, -1.610032558441162, -0.6647781133651733, 0.4975318908691406, -0.8453249931335449, -1.7703320980072021, 1.7428946495056152, 1.136434555053711, 1.3538479804992676, -0.14642371237277985, -0.07694466412067413, 1.7709381580352783, 0.21496552228927612, 0.9746822714805603, 0.9079504609107971, 0.2651413083076477, -0.5099958777427673, -0.2869056463241577, -0.36641138792037964, 0.7782705426216125, 0.9435924887657166, -2.5024802684783936, 0.34916266798973083, 0.9879421591758728, -0.8455316424369812, 1.3763693571090698, -1.166626214981079, 1.1639714241027832, 0.48164859414100647, -0.07214102149009705, -0.506977379322052, -0.29699626564979553, -0.4981561601161957, 1.7730282545089722, 1.5255547761917114, 0.5397740006446838, 0.7835322022438049, 1.0646523237228394, 0.27247509360313416, -0.7745960354804993, -0.19278180599212646, 1.059829592704773, 0.2026078999042511, -0.6296629905700684, 0.1907017081975937, 0.43776172399520874, -0.4080609977245331, 2.5876853466033936, 0.5943648219108582, -1.9728431701660156, 0.3609021008014679, -0.7538818120956421, -0.18157707154750824, 0.2555141746997833, 0.9597761631011963, -0.6579241752624512, 0.21569640934467316, -0.16613611578941345, -0.3796197175979614, 1.080039620399475, -0.0682513564825058, -1.541533350944519, -1.4997137784957886, -0.2529263496398926, -0.22673237323760986, 0.5035735964775085, 0.6436243653297424, 0.06638623028993607, -1.3619771003723145, -0.27226728200912476, -0.9041834473609924, 0.39937448501586914, -0.8438131213188171, -0.19621942937374115], [-0.762913703918457, 2.433061361312866, 0.17800289392471313, 0.46162956953048706, 0.722044050693512, -1.3612020015716553, -0.3151698708534241, 1.5365086793899536, -0.18071149289608002, 0.02399839274585247, -0.7288482785224915, 0.812875509262085, -0.78110671043396, -1.3733820915222168, 0.8047616481781006, -0.616592526435852, -0.09234549850225449, 1.0119127035140991, -0.9925715923309326, -1.060295581817627, -0.13359801471233368, -1.370376467704773, 0.879630446434021, 0.38150089979171753, -0.5585959553718567, -0.35461822152137756, -1.1281135082244873, 0.09009704738855362, 0.2647920548915863, -0.754040002822876, -1.1307857036590576, 0.41044068336486816, -1.1685051918029785, 0.3213596045970917, 0.37881794571876526, 0.7796860337257385, -1.2330378293991089, 0.37253671884536743, -1.2646299600601196, -0.3470182716846466, -0.3230239450931549, -0.7715299129486084, -0.7845309972763062, 0.830390214920044, -1.7276867628097534, 1.5253627300262451, 0.4557819664478302, 0.06195218116044998, -2.8428354263305664, 0.020313745364546776, -0.9940013289451599, -0.7932396531105042, 0.9603474140167236, -0.47567126154899597, -0.4078527092933655, 0.39330974221229553, -1.0249557495117188, -0.6888086795806885, -1.3745850324630737, 1.7667005062103271, 2.48405122756958, -1.19524085521698, 0.05207596719264984, -0.9644792675971985, -0.7553430199623108, 0.3729783594608307, -0.6710032224655151, 0.889775812625885, -1.719411849975586, 1.8444432020187378, 1.0436151027679443, 0.018787285313010216, -0.33988842368125916, -1.0768647193908691, 0.46596646308898926, -0.32741621136665344, 0.49247562885284424, 0.4725164473056793, -0.520545482635498, -1.1314655542373657, -1.3809081315994263, -0.5189192891120911, -1.9685566425323486, -0.21236646175384521, -0.9299747347831726, 0.728118896484375, -1.234112024307251, 0.08939728885889053, -0.7444292902946472, -0.9478126168251038, 1.7703684568405151, 0.06917912513017654, -0.11800269782543182, 0.046414636075496674, -2.0811500549316406, 0.8942763805389404, -0.07817277312278748, 1.3934911489486694, 2.041721820831299, -0.5655545592308044, 2.0656962394714355, -1.3376824855804443, -1.045270323753357, -0.16772282123565674, 0.14876237511634827, -0.0470268614590168, -0.6348888278007507, 0.470316618680954, -1.4055017232894897, -0.9958858489990234, -0.5323649048805237, 1.0834496021270752, 0.4376561641693115, -0.4126680791378021, 2.2134904861450195, -0.35388049483299255, 0.7019833922386169, 0.8059570789337158, -1.8056637048721313, 2.0405595302581787, 1.5777438879013062, -1.2250686883926392, 1.4868721961975098, 0.9804887175559998, 0.4948561191558838, -0.8227676153182983, -0.7519480586051941, 0.31780585646629333], [0.7810820937156677, -1.2410439252853394, -1.5102925300598145, 0.19840866327285767, 0.5692209601402283, -0.6847190856933594, 0.2403387576341629, -1.3229308128356934, -0.9179531335830688, -1.134426236152649, 1.7433744668960571, -0.6588704586029053, -0.37798017263412476, 0.13091248273849487, 0.1541386991739273, 0.5260553359985352, 0.8917555809020996, -0.36729443073272705, -0.2336275428533554, -0.10830363631248474, 0.7546058893203735, -0.690161406993866, 1.8877341747283936, -0.2273956835269928, -0.2753303050994873, -1.8660038709640503, -0.8619707226753235, -0.5702549815177917, 0.5885300040245056, -0.4261312186717987, 2.4871721267700195, -0.36906731128692627, 0.8204531073570251, -1.2779076099395752, -0.017933949828147888, 0.1996428370475769, -0.6807484030723572, 0.43846601247787476, 0.6271030306816101, -0.32009652256965637, -1.134860873222351, -1.2111133337020874, 0.9036523699760437, 0.10663734376430511, 0.3833441734313965, -1.469897985458374, -1.8236677646636963, -0.6909236311912537, -0.1294577568769455, -1.0327686071395874, -1.0203129053115845, -0.03393439203500748, -0.9652939438819885, -0.03420160338282585, -0.4931151270866394, 0.3783940374851227, -1.099614143371582, 1.6975940465927124, 0.4857456386089325, 1.0022609233856201, 0.4905641973018646, -2.1441564559936523, -0.08612719923257828, -0.4619148373603821, -0.23634815216064453, 0.21285560727119446, -0.7672753930091858, 0.10741448402404785, -0.6975666284561157, -0.060180407017469406, 0.2861591577529907, 0.3624292016029358, -0.04609519988298416, 1.9443238973617554, 0.9589350819587708, 1.271386742591858, 0.40733423829078674, -2.0341358184814453, 1.6413356065750122, -0.05452103912830353, 1.1465504169464111, 0.4161701202392578, 1.5666345357894897, -1.0401984453201294, 0.2708822786808014, 0.05971657857298851, 1.0458660125732422, 1.307065725326538, 0.08117056638002396, 0.3962325155735016, 0.9633915424346924, 0.5646577477455139, -0.011553332209587097, -1.320554494857788, 0.9217587113380432, -0.8346985578536987, 0.38037651777267456, 0.7353373169898987, -0.213909313082695, 0.6314454674720764, -1.0697826147079468, 0.4158903658390045, 0.8065072894096375, -1.3987839221954346, 0.04837329685688019, -0.7196446061134338, -0.8858428001403809, -0.6636173129081726, 0.10541458427906036, 0.9817190170288086, 2.3574295043945312, 0.795529842376709, -0.7975704669952393, -0.3508766293525696, -0.5401731133460999, -0.13269416987895966, 0.43809774518013, 1.1884647607803345, 0.982755720615387, -0.7221661806106567, 0.2649257183074951, -0.36192113161087036, 1.0002871751785278, -1.4185727834701538, -0.7689863443374634, 1.0263440608978271, -0.8665812611579895, 0.36249953508377075], [-0.3940330445766449, 0.27108529210090637, 0.5450748801231384, -0.7653968334197998, 1.4865329265594482, -0.6025406718254089, 0.7454151511192322, 1.342653512954712, 0.4044193625450134, -0.41291341185569763, -0.7704535126686096, 0.14782634377479553, -0.6547902226448059, -1.5314087867736816, -0.45420730113983154, -0.4829866886138916, 0.10278815776109695, -0.2911914587020874, 1.5196501016616821, -0.5462928414344788, 0.8826791644096375, -1.5563257932662964, -1.5813117027282715, -0.21565833687782288, 1.013563871383667, 0.6995795965194702, -1.2804335355758667, 3.1605842113494873, -0.7002931237220764, -0.593912661075592, 0.34467563033103943, -1.8128859996795654, -0.23423069715499878, -1.705671787261963, -0.23964639008045197, -1.8231539726257324, -1.4242632389068604, 0.49569815397262573, 0.5871658325195312, 0.631659984588623, -0.5487908124923706, -0.2749413251876831, -0.6478532552719116, -0.6021860241889954, -0.8284935355186462, 0.0409311018884182, -0.408908873796463, -0.6093286871910095, -0.2088998258113861, -0.7536783814430237, 0.32207441329956055, -1.371214509010315, 0.5386488437652588, 0.21736590564250946, -0.39990583062171936, -1.778836727142334, -0.6789687871932983, -0.14119715988636017, -0.19592706859111786, -2.011474370956421, -1.438939094543457, -0.6172282099723816, -0.2037373036146164, -0.637346088886261, 1.1614280939102173, -1.0111945867538452, -2.3373560905456543, -0.4467787444591522, 1.2424980401992798, 2.0092694759368896, -0.45099174976348877, -0.9375317096710205, -0.39134329557418823, -1.2346553802490234, -0.04395626485347748, -1.6089929342269897, 0.5124753713607788, 0.24071353673934937, -0.08250148594379425, -0.9863521456718445, -1.4388281106948853, 0.34639954566955566, 1.487364411354065, 1.9713503122329712, -1.046692967414856, 0.5072612762451172, 0.9455006718635559, 1.3754316568374634, 0.016682937741279602, -0.9727343320846558, 0.20658664405345917, 1.8018969297409058, -0.1690698266029358, 0.8621377944946289, -0.36315011978149414, 2.4331655502319336, 0.2961371839046478, -1.4339208602905273, -0.36382225155830383, 1.3938872814178467, -0.02478072978556156, -0.1713947355747223, 0.39764925837516785, -0.5365879535675049, 0.4950653612613678, -0.7034848928451538, 0.9929757714271545, 0.727816641330719, -0.2351597547531128, 1.2302827835083008, -0.490774542093277, -0.02484486624598503, -2.174983024597168, -1.4391669034957886, -0.06990768760442734, 1.2051821947097778, -0.8615769147872925, -0.6858619451522827, -0.6423400044441223, 1.3784235715866089, -0.4173488914966583, 0.5954895615577698, -0.07803013175725937, -1.033268928527832, 1.0691864490509033, -1.3434323072433472, -0.5442119240760803, -1.44990873336792], [-0.12550562620162964, 0.824407696723938, 0.6232852935791016, -1.293001413345337, 1.9253735542297363, 1.576488971710205, -0.5927104353904724, -0.01348799653351307, 0.02124762535095215, 1.357673168182373, 1.2116039991378784, -0.40165773034095764, 0.9490537047386169, 0.20647403597831726, -1.862677812576294, -1.1352397203445435, 0.4760999083518982, -0.20310385525226593, 0.6573805809020996, -0.5814955234527588, -0.5713973641395569, -0.9215046167373657, -1.1234705448150635, 0.10054556280374527, 0.8591082096099854, -0.3047860562801361, 0.15961787104606628, -0.24228383600711823, 0.17185139656066895, -1.5488277673721313, 0.9721858501434326, -0.8733809590339661, -0.3204267919063568, -0.815991222858429, -1.8626168966293335, 1.0641162395477295, 2.0861828327178955, -1.456270694732666, 1.2069454193115234, -1.0930171012878418, -1.305781602859497, -2.122091054916382, 0.7572465538978577, -1.1759467124938965, -0.5596474409103394, 0.887330174446106, 0.49118876457214355, 1.3701658248901367, 3.0175232887268066, -1.7373175621032715, -0.9926794171333313, 0.7183748483657837, -0.4009222388267517, 1.069179892539978, -0.5370411276817322, -1.4854332208633423, 0.9985514283180237, -0.6382997632026672, 1.7883185148239136, 0.95782870054245, -0.11599108576774597, -0.3312135934829712, 0.4399425983428955, -2.122002124786377, -0.06094982847571373, 0.6121554970741272, -0.50218266248703, -0.191621795296669, -0.05481914430856705, 0.7789705395698547, -0.4525114893913269, 0.34378984570503235, 0.4071781039237976, 1.2994251251220703, -0.5617650151252747, 0.41126707196235657, 0.4274110198020935, -1.0390411615371704, 2.354135274887085, -3.5219836235046387, -0.27354517579078674, -0.28708377480506897, -1.0754282474517822, -2.1705949306488037, -0.447051465511322, -1.1147209405899048, 0.4322066009044647, -0.7716111540794373, -0.4538106620311737, -0.4399680495262146, 0.09290647506713867, 0.29573479294776917, -1.5549216270446777, -0.184552863240242, -0.5431607961654663, -0.15610778331756592, 0.911483645439148, 0.8907297849655151, 0.38306382298469543, -0.4022919237613678, 0.07262211292982101, -1.6127992868423462, 0.46992918848991394, -0.5476841926574707, 0.9783614873886108, 0.2355899065732956, 0.033674366772174835, 2.2080094814300537, 1.019826054573059, -0.947381317615509, -1.0639443397521973, -0.25561314821243286, 1.803406000137329, 0.038702260702848434, 2.4994258880615234, 1.0876396894454956, 0.3856733739376068, -0.5119537115097046, 1.4292645454406738, 0.9127357602119446, 0.17945201694965363, 0.21104896068572998, -0.5404210090637207, 0.32564541697502136, 1.4050960540771484, -0.3530493676662445, -0.3354184329509735, -1.9241511821746826], [-0.706093430519104, 1.2550404071807861, 0.3062547743320465, -0.11224211752414703, -0.15663069486618042, -0.9208134412765503, 0.027426807209849358, 1.5539484024047852, -0.42955973744392395, -1.9108433723449707, 0.30510467290878296, -0.18325282633304596, -0.5657311081886292, 1.6401768922805786, 1.0876764059066772, 1.2716548442840576, 0.31519249081611633, -0.5690072774887085, 0.23161402344703674, 0.4098149240016937, -0.10992449522018433, 1.5773096084594727, -0.7878497242927551, 1.403977870941162, -1.0542237758636475, -0.09424196183681488, -0.879902184009552, 0.08107833564281464, 0.6627103090286255, 0.17059749364852905, 0.38838014006614685, 0.7423406839370728, -0.32965466380119324, -1.5527818202972412, -2.0923709869384766, -0.5898964405059814, 0.4611615538597107, 1.356177806854248, 1.532060980796814, 1.3984910249710083, -0.9149269461631775, -1.280517578125, 0.6604238748550415, 0.4887317717075348, -1.3349740505218506, -0.6540929675102234, 1.5570589303970337, 0.3683772087097168, -0.05262456461787224, 1.8083899021148682, 0.9620932340621948, 0.8181843757629395, -0.44414716958999634, 0.04635591432452202, 0.626778244972229, 1.440873146057129, 2.1701064109802246, 0.6182786226272583, 0.16829949617385864, -1.677838921546936, -0.5072048306465149, 2.1079630851745605, 0.2503341734409332, 0.6389470100402832, -0.2984638810157776, -0.6027877330780029, 0.6643354892730713, 2.341399908065796, -0.4313699007034302, -0.7810139060020447, 0.3199141323566437, 1.184971809387207, -1.2096761465072632, -2.7447686195373535, 0.36196368932724, 0.5180068612098694, -1.1644704341888428, 0.8824573755264282, 0.765149712562561, 1.257925271987915, -0.143232062458992, 0.06001749634742737, -0.2970748543739319, -0.3119240999221802, -0.2798691391944885, -1.9572365283966064, 1.1823030710220337, -0.6555652022361755, -2.0649001598358154, 1.6585417985916138, 0.8264063596725464, 1.0924968719482422, 0.053400974720716476, 1.008695125579834, -1.1818304061889648, -0.3528427183628082, 0.9811991453170776, 0.47642311453819275, 0.380883127450943, -0.641837477684021, 1.067899227142334, 1.3074103593826294, -1.112310528755188, 0.74159175157547, -1.5983396768569946, 1.6161038875579834, -0.15749818086624146, 1.4457347393035889, 1.9494389295578003, -0.034164898097515106, 0.14052550494670868, -0.06720951199531555, 0.580222487449646, -0.31124863028526306, -0.5552718639373779, 2.024448871612549, -0.26624876260757446, 0.7607421278953552, -1.2176902294158936, 1.0539510250091553, -0.6655125021934509, -0.5837457180023193, -0.16567851603031158, 0.5178366899490356, 1.4065860509872437, 1.4129215478897095, 0.2219710648059845, -0.8504213094711304], [-0.22937525808811188, -0.031123194843530655, -0.055857107043266296, 0.21047967672348022, 0.9611580967903137, 2.234551191329956, -0.1838817149400711, 0.7384316921234131, 0.026146981865167618, -1.6357903480529785, -1.2840211391448975, 0.6176198720932007, -1.4593727588653564, -0.2098316103219986, 1.1876863241195679, 0.6429346799850464, -0.5395729541778564, -0.008836508728563786, -1.2228063344955444, -0.24795305728912354, 0.6503553986549377, -1.4256232976913452, 0.5615832209587097, -0.3153071403503418, -1.2992898225784302, 1.0889893770217896, 0.07803076505661011, -0.31920233368873596, -1.5192937850952148, 0.3481908142566681, -2.215482234954834, 1.1557879447937012, -0.5346298217773438, 0.6584802865982056, 0.6010912656784058, -0.66454017162323, 0.2986738979816437, -0.44424617290496826, -1.1085931062698364, -0.44855883717536926, -1.1262568235397339, -0.228068009018898, 0.1576799601316452, -0.8165335655212402, -0.1820629984140396, 1.820281982421875, 0.5505698919296265, -0.8430696725845337, 0.5476015210151672, 0.0947728306055069, 1.4444105625152588, 0.28812462091445923, -0.6209930777549744, 0.11789995431900024, 0.02130158804357052, -1.2251816987991333, 1.2836378812789917, -1.7257939577102661, 0.3767150938510895, -0.6631196141242981, 0.32816246151924133, -0.9691484570503235, 1.8824623823165894, -0.362898588180542, -0.4394547939300537, -0.6283766627311707, -0.14986498653888702, -1.0145827531814575, -0.7844410538673401, -1.393104910850525, 0.27019575238227844, -0.27518758177757263, 0.917082667350769, 2.478748083114624, -1.3233318328857422, -1.1581088304519653, 0.8870998620986938, -0.13191869854927063, 1.3562253713607788, 1.6705775260925293, -1.791121244430542, 0.30595701932907104, 1.5744013786315918, -0.6627724170684814, 0.4029735326766968, -0.6727270483970642, -0.1779399961233139, 1.357326865196228, 0.5597015023231506, 0.4183037281036377, 1.0682486295700073, -2.447343349456787, -0.37726372480392456, 2.046715497970581, -0.02888103947043419, -0.1950983703136444, -0.19311657547950745, -0.07306718081235886, -1.7268187999725342, 0.5532025694847107, -1.332375168800354, 0.08652910590171814, -1.790600061416626, -0.949612557888031, 0.7657254338264465, 0.003186602611094713, -1.3498494625091553, -0.5259572863578796, -0.3331679105758667, 1.8878159523010254, 1.1131558418273926, 1.1645824909210205, -1.8494517803192139, 0.1464487612247467, 0.6643325090408325, 1.184160590171814, -0.7967758178710938, 0.5356060266494751, -0.69825279712677, -1.3194807767868042, -0.05202692002058029, -0.20720352232456207, 0.9486193656921387, -1.7991572618484497, 1.4323859214782715, -0.67196124792099, 0.3060544431209564, 0.1665516346693039], [-0.7495365738868713, 0.289816677570343, 0.2763743996620178, -0.9167601466178894, -0.06960836797952652, 0.6875860095024109, 2.0262627601623535, 0.023069560527801514, 0.00592839764431119, -0.011107479222118855, 1.1628835201263428, 1.4627048969268799, -2.0138039588928223, 0.6218153834342957, 0.16448424756526947, 0.8437315225601196, 0.9070429801940918, 0.48382502794265747, 1.1046441793441772, -1.039840817451477, 0.4912317991256714, -0.009657252579927444, 1.0827234983444214, 2.7579500675201416, 0.08270761370658875, -1.5380656719207764, -0.8046492338180542, -0.1735735386610031, 0.6292732357978821, -0.36397555470466614, -0.5309457182884216, -0.3532108962535858, 1.4162640571594238, 2.009626865386963, 0.7831499576568604, -0.77842777967453, 0.13404904305934906, 0.038340967148542404, -0.6491419076919556, -0.667774498462677, 0.8871328234672546, 0.2961488664150238, 0.7498253583908081, -0.29852423071861267, 0.5741457939147949, 0.9322463870048523, -0.5296760201454163, 0.8260870575904846, -0.25895535945892334, 0.26911741495132446, -1.0507246255874634, 0.5621358156204224, -0.6125735640525818, -0.19398662447929382, 1.7742259502410889, 2.5579352378845215, -0.9168949723243713, -0.7850245237350464, 0.30661794543266296, -0.401475191116333, 0.08200126886367798, 0.37024542689323425, 0.31795021891593933, 1.4296648502349854, -1.52814781665802, -0.4767799377441406, 0.0327485166490078, 2.0761234760284424, 0.75562584400177, -0.6791477799415588, -1.1953787803649902, 1.5224689245224, -0.31244325637817383, -0.33280161023139954, 0.01546926237642765, -2.8689053058624268, -0.6836438179016113, 2.0227744579315186, -0.3913925290107727, 1.5883028507232666, 2.152780771255493, -0.32772019505500793, 0.9175805449485779, 0.21361036598682404, -0.4049994945526123, -0.9937410950660706, -1.0420360565185547, 1.1010117530822754, -0.4243997037410736, -2.5697004795074463, -0.7223507165908813, -0.9854602813720703, -1.11403489112854, 1.3373929262161255, 2.707960605621338, 0.7765424847602844, -1.5810515880584717, 0.7110681533813477, 1.0449658632278442, -0.7678543329238892, 0.7609748840332031, -0.8403792381286621, -0.9949334859848022, -0.06835244596004486, 0.2205619215965271, -0.16469286382198334, -1.8071871995925903, 1.0431060791015625, -0.2586037516593933, -1.1233512163162231, -0.4809536337852478, -1.8237900733947754, 1.0268901586532593, -0.7550472021102905, 1.1682747602462769, -0.02460523135960102, -1.3756828308105469, 1.0934780836105347, 1.5033140182495117, 0.033240098506212234, -0.5758990049362183, 2.6671082973480225, 0.7290412783622742, 1.6590434312820435, 1.4265378713607788, -0.3463164269924164, -0.05815422534942627, 0.726638674736023], [-0.39816775918006897, -0.849320650100708, 0.8414976000785828, 2.098839521408081, -0.12460994720458984, 0.5370938777923584, -0.7095720767974854, -1.2264683246612549, -0.42764344811439514, 0.7801768779754639, 1.5673774480819702, 1.5305590629577637, 0.22669728100299835, -0.5828291177749634, 2.002678632736206, 0.5253174304962158, 0.9385426640510559, 0.38539719581604004, -2.0780556201934814, -0.3050336241722107, 0.039529234170913696, 0.8700768351554871, -0.461709588766098, -0.7222170829772949, 1.279605507850647, 0.4433504641056061, -1.0816447734832764, -0.14330436289310455, 0.9716112017631531, -0.07251255959272385, 2.969219446182251, 1.2774500846862793, 0.7318170070648193, 0.4627310335636139, -0.18781137466430664, -1.163831353187561, -0.7399178147315979, -1.6000555753707886, -1.3057518005371094, -2.3739230632781982, -0.3964286744594574, 1.1944491863250732, -1.4067344665527344, -0.28580713272094727, -0.6470693349838257, 0.8736305236816406, -0.06918126344680786, -1.4539422988891602, 0.24300408363342285, -0.8501731157302856, 0.1059015616774559, 0.10583601146936417, -0.21552689373493195, 0.4250901937484741, -0.10084354132413864, 0.061429280787706375, -0.06581419706344604, 2.2649662494659424, 1.410439372062683, -0.33622944355010986, 0.316849946975708, -2.0786943435668945, -0.7840953469276428, -0.7776125073432922, -0.015456503257155418, 0.1043367013335228, -0.27672332525253296, 0.7741432189941406, 1.3498826026916504, 0.5815457701683044, 0.14102216064929962, -2.022839307785034, -0.11508037894964218, 1.5725913047790527, -2.5925745964050293, -0.6693695783615112, -0.7526468634605408, -0.2548757493495941, -1.4466373920440674, 0.5951774716377258, 0.5952891111373901, -0.1672644466161728, 1.1334940195083618, 0.45997554063796997, -0.8544591069221497, 0.1401507705450058, 1.4629932641983032, 0.5273640155792236, 1.4851300716400146, -0.9982955455780029, 1.6703702211380005, 0.30329832434654236, -0.31725946068763733, 0.2964245676994324, 0.987092912197113, -0.566332221031189, 2.2341439723968506, 0.6546227931976318, -0.7128633260726929, -1.6095434427261353, 2.1755707263946533, -0.5261672735214233, 1.3617178201675415, -0.14012350142002106, -0.04829215258359909, -0.5233706831932068, 0.6167672276496887, -0.332250714302063, -0.7031364440917969, -0.4012520909309387, -1.3763806819915771, 0.29796913266181946, 0.5108318328857422, 0.9496219754219055, 0.36471131443977356, 0.5025008916854858, -1.6505968570709229, 0.5214338302612305, -0.5037803649902344, 0.45566481351852417, -0.09024263918399811, 0.17979919910430908, 0.5218713879585266, -0.4565061330795288, -0.9370886087417603, -1.6470407247543335, -0.6794797778129578, -1.1741845607757568], [-0.6863161325454712, -0.021700862795114517, 0.0036106056068092585, 2.239511489868164, -0.07535591721534729, 0.20591622591018677, -0.19648267328739166, -1.9488716125488281, 0.5183541774749756, -1.140721321105957, -0.9710555076599121, 0.6571006178855896, -0.2671258747577667, 0.5487292408943176, -1.7726083993911743, 0.050362635403871536, 0.6649503707885742, 0.8765999674797058, 0.4212053418159485, -0.010817183181643486, 0.9766098260879517, 0.8991041779518127, -1.358541488647461, -0.42628014087677, -0.40880414843559265, -0.6499050855636597, 0.20707134902477264, -0.05669603869318962, 1.0538005828857422, 1.4908251762390137, 1.494193434715271, -0.2211911380290985, 0.9678504467010498, 0.5658203959465027, -0.20149964094161987, -0.5015381574630737, 0.8703970909118652, 1.894789218902588, -1.1221531629562378, -0.34775498509407043, 0.33576157689094543, 0.5628989934921265, -0.7988929152488708, 1.0789715051651, 0.45142316818237305, 0.11252794414758682, 0.3927858769893646, 1.7177584171295166, -0.2836470603942871, 0.4480361044406891, -0.646460771560669, -0.9122582674026489, 0.436991423368454, -0.49270153045654297, -0.6102678179740906, 1.1857917308807373, -1.150403618812561, 0.8462603092193604, -0.22206807136535645, 0.020692894235253334, -0.7317925095558167, -0.6211466789245605, -0.025716004893183708, -0.10205067694187164, -0.6265857219696045, -0.33376815915107727, -0.3868946433067322, -0.06713210791349411, -1.6701136827468872, 0.7656205296516418, -1.1339083909988403, 0.4726611077785492, 0.45040255784988403, 0.9887039661407471, -2.161024808883667, -0.49678879976272583, -1.8777601718902588, -0.5913706421852112, 0.1589280664920807, 1.1400922536849976, 0.7695087790489197, 0.06250006705522537, 0.9171642661094666, -0.5148950219154358, 0.5548708438873291, 0.3824104368686676, 0.04703545942902565, 1.1969764232635498, 0.7296481132507324, -0.7196521162986755, 0.206041619181633, 1.1398513317108154, 1.0356656312942505, 0.4333835542201996, -0.9897101521492004, 0.37752506136894226, -0.42153531312942505, 1.3778973817825317, -1.2679373025894165, 1.262074589729309, -0.22536517679691315, -1.2761515378952026, -2.1908817291259766, 0.8301591873168945, -0.1289365440607071, 0.30691495537757874, 0.42206019163131714, 0.7420799136161804, -2.8319122791290283, 2.3600223064422607, -0.7041349411010742, 0.8941081166267395, 0.2991653382778168, 1.8705637454986572, 0.2985823154449463, 0.624316930770874, -0.8784590363502502, -1.0280957221984863, 2.3089187145233154, 0.3470971882343292, 0.02226690948009491, 1.5679458379745483, -0.3593596816062927, 1.8859163522720337, -1.0327335596084595, 1.2932730913162231, -0.615908682346344, 0.20397014915943146], [-0.09379197657108307, -1.9962724447250366, 0.4852420389652252, -0.1755007654428482, 0.835622251033783, 0.8454358577728271, -1.6879476308822632, -0.5927635431289673, 0.12542565166950226, 0.8858105540275574, 0.28362104296684265, -1.2422016859054565, -0.7383767366409302, 1.3007224798202515, 1.335720419883728, 0.5599468350410461, 0.5923939943313599, -0.6752468347549438, 0.7869361042976379, 1.3012747764587402, -3.203237771987915, 1.8435481786727905, 0.6395443081855774, 1.0216476917266846, 0.9727802872657776, 1.6482821702957153, -0.37739402055740356, 0.16983076930046082, 1.1774457693099976, -0.6941146850585938, -0.23756223917007446, 0.6269533634185791, 1.3157665729522705, -0.5274592638015747, -1.7032431364059448, -1.1688110828399658, -0.5192530155181885, 1.635398268699646, -0.09292635321617126, -0.7362163662910461, -0.5585469007492065, 0.9580547213554382, -1.759476661682129, -0.15172983705997467, 1.556337833404541, 0.3121704161167145, -1.275650143623352, 0.7143244743347168, 0.32555651664733887, 1.5471410751342773, 1.2855989933013916, -1.5267658233642578, -0.3477105498313904, 1.7959555387496948, -0.21370598673820496, 1.2310082912445068, -0.06757145375013351, -0.6780627965927124, -0.29660478234291077, -1.975098729133606, 0.6330953240394592, -0.8134247660636902, 0.27396708726882935, -0.14131225645542145, 0.9129247665405273, -1.1480342149734497, -0.3047024607658386, -0.5717043280601501, -0.7499653697013855, 0.49007448554039, 1.6845905780792236, -0.10517869889736176, -0.46083545684814453, -1.9889216423034668, -0.026667168363928795, 0.7675496339797974, 0.2194749414920807, 1.1821492910385132, 0.45800521969795227, 0.4403729736804962, -2.351790189743042, 0.04289468750357628, 0.17814457416534424, 0.0028596112970262766, -0.7443169951438904, 1.1541211605072021, 0.09270833432674408, -1.4914225339889526, 0.9435726404190063, -0.3575683832168579, -0.5981990098953247, -1.6703181266784668, 0.4457336664199829, -0.007918172515928745, 0.9259700775146484, -0.8058423399925232, 0.010323366150259972, -0.013198466040194035, 0.4141002595424652, 0.824140191078186, 1.1332215070724487, -0.25074082612991333, 0.8497443795204163, 0.3573383092880249, 1.9920235872268677, 3.6708974838256836, 2.301109552383423, 0.7281401753425598, -0.6842353940010071, -1.3626073598861694, 1.1034801006317139, -1.5692020654678345, 1.762819766998291, -0.7316243648529053, 2.306032419204712, 0.9209544062614441, -1.354882836341858, 0.47861114144325256, 0.09605682641267776, -1.3426491022109985, -0.22557437419891357, 1.6058179140090942, 0.7172946929931641, 0.4558175802230835, 0.2295762598514557, -0.8795448541641235, -0.035411275923252106, -0.6628972291946411], [-0.8465783596038818, 0.24945765733718872, -0.048065874725580215, -0.16271238029003143, -0.9329465627670288, -1.50546395778656, -1.0274535417556763, -1.7452492713928223, -0.1862277090549469, -1.2039557695388794, -0.09704840928316116, 0.12952877581119537, -0.8579588532447815, 0.7488943934440613, 0.1974693089723587, -1.8467317819595337, -0.6272196173667908, 1.9912629127502441, -1.8595236539840698, 0.9406784772872925, -1.8933404684066772, 0.0006738129304721951, -0.7712734341621399, 1.0977615118026733, -0.7713631987571716, -0.7689166069030762, -1.07658851146698, -0.37754297256469727, 1.9560024738311768, -0.48359137773513794, -0.7872973680496216, -1.0149980783462524, 0.30431365966796875, -1.9439520835876465, 0.1640835702419281, 1.973632574081421, -1.5669431686401367, 1.2765884399414062, -1.3604516983032227, -0.9313194751739502, 0.585989773273468, -0.6315059065818787, -1.5710773468017578, -0.28693071007728577, -0.46974262595176697, 0.32769638299942017, -0.3454076051712036, -1.6719568967819214, 0.42452889680862427, -0.13848334550857544, -0.5476919412612915, 1.3024789094924927, -0.7762354612350464, 0.9347951412200928, -1.8446663618087769, 1.8347359895706177, 0.3440703749656677, 0.6740133166313171, -1.2006558179855347, 0.3029960095882416, -2.156404972076416, -2.261711835861206, -0.3474273383617401, 1.5284382104873657, 1.997449517250061, 0.058534979820251465, -1.5007675886154175, -1.418161153793335, 0.820441484451294, 0.45278412103652954, -1.0277142524719238, -0.4529785215854645, 1.444018840789795, 0.6619062423706055, 0.0020270184613764286, -0.5482857823371887, -0.2472078651189804, -0.70097416639328, -1.0118986368179321, 0.2145083248615265, -1.5016485452651978, -0.24583405256271362, -1.774051308631897, 0.13164615631103516, -0.9821722507476807, -0.018052970990538597, -0.7760834693908691, -0.7935580015182495, 1.3284268379211426, 0.8747313022613525, 1.2285817861557007, 0.4539274573326111, 0.21644224226474762, -0.31615185737609863, 0.7472224235534668, -0.7760223150253296, 0.10025472939014435, -1.5554007291793823, -2.093536138534546, -0.43968093395233154, 0.3808366060256958, 1.7601546049118042, -0.7605502605438232, -0.8153784275054932, -0.40336692333221436, 1.2422981262207031, -0.524910032749176, -1.7048438787460327, 0.9412360787391663, 0.2509426772594452, -0.0528528057038784, 0.5206910967826843, -0.7987545728683472, -0.8904469609260559, 0.019277483224868774, -1.8594359159469604, -1.1499747037887573, 1.7308554649353027, -1.260978102684021, 1.174919843673706, 0.9518308639526367, -0.6649860739707947, 1.7360767126083374, 0.9505337476730347, 0.5832464098930359, -2.3668386936187744, 0.12669618427753448, -1.6614000797271729], [0.3601112961769104, -1.1717687845230103, -0.477131724357605, -0.2936362624168396, -0.9471578001976013, 0.1616765558719635, 2.3883416652679443, -0.1354706734418869, 0.039187636226415634, 0.8468939065933228, 0.224545419216156, -0.134109228849411, 0.9447107911109924, 0.7764072418212891, 1.0801286697387695, 0.3350701630115509, 0.18133501708507538, -0.9406602382659912, 2.2844743728637695, 0.642258882522583, 0.28954586386680603, 1.1975756883621216, -1.3091013431549072, -0.3640930950641632, 0.09708569943904877, -0.5056155920028687, 0.18222743272781372, -0.05597569793462753, -1.2773103713989258, 1.4349323511123657, -0.21217729151248932, 0.25705215334892273, 0.5949111580848694, 0.39105531573295593, -0.08136176317930222, 1.2677114009857178, -0.10271012783050537, -0.47234398126602173, 0.03608211502432823, -1.004886507987976, -2.48958420753479, 0.5853361487388611, -1.6785424947738647, -0.06460748612880707, -0.2133025974035263, -1.9145615100860596, 1.902742624282837, -2.263813018798828, -0.045523449778556824, 0.30732572078704834, -0.10490109771490097, 1.2985811233520508, -0.806641697883606, 0.479373574256897, 0.44265156984329224, -0.05974826589226723, 1.7291184663772583, 0.6648095846176147, -0.9263502955436707, 2.1105546951293945, -1.2967674732208252, -1.0469486713409424, 0.05638578161597252, -1.893907904624939, -0.8261733055114746, -1.54417884349823, 0.9808646440505981, 1.069329857826233, -0.03528639301657677, 0.2840830683708191, -0.889116644859314, 0.47788873314857483, -0.33163055777549744, 1.4744642972946167, 0.36948180198669434, -0.7050499320030212, -0.23137152194976807, 0.855002760887146, 0.9579254984855652, 0.9432278275489807, 0.7416967749595642, -0.22306139767169952, -0.33007776737213135, 1.582944631576538, 0.6047495603561401, 0.14844778180122375, 1.0138959884643555, -0.5454587340354919, -0.7347758412361145, 0.12027478963136673, 0.5618093013763428, 0.9684557914733887, 0.3577098846435547, 0.5332005023956299, -0.9751952886581421, -0.8870190382003784, -0.23523865640163422, -0.6409240961074829, -2.0343544483184814, 0.29156404733657837, -0.2940213978290558, -0.7295416593551636, 0.04358135908842087, -0.7889358401298523, 1.042988896369934, 0.4948394298553467, 0.5875096917152405, -0.2660442888736725, -0.260845422744751, 0.9420420527458191, -1.8525395393371582, -0.4361816942691803, 1.2037557363510132, -0.2658795416355133, -1.3686293363571167, -0.913016140460968, 0.8017066717147827, 2.357081890106201, -1.5741852521896362, -0.011037669144570827, -1.3713434934616089, -0.4725058674812317, -0.17330287396907806, 0.5187586545944214, -0.6825974583625793, -1.0152028799057007, -0.11502595990896225, -2.292706251144409], [-1.2661347389221191, -0.18478858470916748, 0.9615389108657837, 0.06698257476091385, 1.6129900217056274, -1.4127836227416992, -0.3237355351448059, -0.9411302208900452, -0.7020367383956909, -0.3010305166244507, 0.3383689224720001, 0.656032145023346, 0.9874903559684753, -1.0900964736938477, 0.975756049156189, 0.018958067521452904, -1.2633548974990845, 0.7235466241836548, -0.39963576197624207, -1.2380714416503906, 0.521102786064148, -0.29251599311828613, -0.7767835259437561, -0.3592793643474579, -0.4728417992591858, 1.2142106294631958, -0.22438813745975494, -0.057474177330732346, -0.7110173106193542, 1.2941672801971436, 0.037555817514657974, 0.04906583949923515, -0.6291754841804504, 1.043479561805725, 0.4585361182689667, 0.619217574596405, 0.17620237171649933, -0.8570483922958374, -0.9070245623588562, -0.3236912488937378, -0.4738878011703491, 0.9825176000595093, 1.3811721801757812, -0.5542489290237427, 2.289698600769043, -0.1523250937461853, 0.151119664311409, -1.869342565536499, -1.2508857250213623, 0.9131284952163696, 1.7629269361495972, -0.15418407320976257, 0.09835685044527054, 1.2944819927215576, -0.9247165322303772, -0.8144465088844299, -0.5163939595222473, 0.4582059979438782, 1.0850303173065186, 2.026435136795044, -0.636783242225647, 0.38057607412338257, 1.8491915464401245, 0.3215418756008148, 1.6216297149658203, -0.10590231418609619, 0.9402409791946411, -0.6235945820808411, -0.7620998620986938, -0.8528894186019897, -0.3151451349258423, -0.2721370756626129, -0.1484392136335373, -0.9271296262741089, -0.3386197090148926, 0.11002840101718903, -1.188326120376587, -0.04137788712978363, 0.3405037522315979, 0.2745721638202667, -0.7223395109176636, 1.5615580081939697, 0.6010091304779053, 0.4147796332836151, 0.15285328030586243, 0.16906534135341644, 0.40314650535583496, -2.2537124156951904, -0.41341927647590637, -0.5335128903388977, 1.3439759016036987, -0.4054950177669525, 2.9203455448150635, -1.2144007682800293, -0.25833165645599365, -1.2553802728652954, -0.2806761860847473, -1.8215667009353638, -0.7340378165245056, -1.0343159437179565, -0.32135531306266785, 0.5569586753845215, -1.8504523038864136, 1.6437420845031738, -0.4225786030292511, -1.8060818910598755, -0.7122315168380737, 0.42280113697052, -0.44362249970436096, -1.6653517484664917, -0.23112644255161285, -0.36446791887283325, -0.5438124537467957, 1.2392386198043823, 2.3471014499664307, 1.6220643520355225, -1.6830518245697021, 0.9030996561050415, -0.5211799740791321, 0.5153630375862122, -0.9264425039291382, 0.2602972686290741, -0.24696922302246094, -1.2664051055908203, 0.30977964401245117, -0.6236593723297119, 0.43303754925727844, 0.5186185240745544], [-1.6051288843154907, -0.10679648071527481, 0.8155761957168579, -2.0844593048095703, -0.9324349164962769, 0.5359466671943665, -0.6645286083221436, 0.7122296094894409, -0.27309831976890564, 1.105035424232483, -1.809510350227356, 1.4582375288009644, -0.003297551302239299, -2.059328317642212, 0.855501115322113, -1.0804756879806519, 2.963817834854126, -0.732227087020874, 1.0847303867340088, -1.503359317779541, -0.7616763710975647, -0.5010174512863159, -0.0634661540389061, -1.6681276559829712, -1.0294655561447144, -1.2968308925628662, -0.8306988477706909, 2.547710418701172, 0.34122434258461, -0.9795224666595459, -0.09182730317115784, 0.6521453261375427, 1.0413398742675781, -0.4675960838794708, -0.4442271292209625, -0.6807997822761536, 1.420650839805603, 0.043629709631204605, 0.45149102807044983, 0.8422252535820007, -2.909269332885742, 0.3515825867652893, 1.7209491729736328, -0.8885916471481323, -0.7677984833717346, -0.5089592337608337, 1.5088133811950684, 1.317800760269165, 0.5476685166358948, -0.5817367434501648, 1.2402281761169434, -0.618073046207428, 0.2001921534538269, 0.9287322759628296, -0.5003971457481384, -1.048296570777893, 0.4446294903755188, 0.8445277810096741, -0.46920841932296753, 0.04112778604030609, 0.02954833395779133, -1.8339664936065674, 2.4129292964935303, -0.10053634643554688, 1.2437583208084106, 0.6672672629356384, 1.0824894905090332, -1.1405338048934937, -0.19218534231185913, 0.718498945236206, 0.6658342480659485, -1.1080944538116455, -0.051495544612407684, 1.0593206882476807, -0.10695692151784897, -0.8847600221633911, 1.0012890100479126, 0.13843423128128052, -0.949749231338501, -0.4549024701118469, 0.9434827566146851, -0.7743890881538391, -1.070935845375061, 2.260831117630005, 0.14314690232276917, -0.11130274832248688, 0.1544804871082306, 0.789095401763916, -1.1215088367462158, 0.6656084656715393, 1.3153774738311768, 0.5708997845649719, -0.3592052757740021, 0.3963399827480316, -2.4781150817871094, 0.6238403916358948, -0.6502695679664612, -0.23471881449222565, -1.0526484251022339, 0.5153329372406006, 0.9534730315208435, -0.5348511338233948, 1.4486687183380127, -2.1276028156280518, 0.5519852638244629, 0.2507448196411133, 0.22759462893009186, -0.7950518131256104, -0.04613832011818886, 1.470175862312317, 0.6920928359031677, 0.0020862456876784563, -0.587225079536438, 0.6228943467140198, -0.019611498340964317, -1.5767306089401245, -0.6233755350112915, -0.2905738651752472, -0.5944334268569946, 0.8138564825057983, 0.03720889613032341, -0.10481703281402588, -2.1818835735321045, -2.428076982498169, 1.5565972328186035, -1.3629670143127441, 0.30602115392684937, 0.20486892759799957], [-0.7208933234214783, 0.18528492748737335, 0.45541900396347046, -0.546308159828186, 0.4907134175300598, 0.28344041109085083, 0.05414154380559921, 0.4454735219478607, -0.6866012215614319, -0.0020109014585614204, 0.1873960942029953, 0.25812795758247375, 0.9168375134468079, 0.06656193733215332, 1.9219306707382202, -1.0100866556167603, -0.6936630010604858, 1.629640817642212, -0.7279607057571411, -0.7686179280281067, 0.7059991955757141, 1.0209912061691284, -0.12035232782363892, 0.4789889454841614, 0.41884833574295044, -0.05736347287893295, -0.5727914571762085, -1.8727738857269287, -0.36583662033081055, -0.27248480916023254, -0.24891553819179535, 1.1366904973983765, -1.0482293367385864, 0.04110381752252579, 0.1570010781288147, -0.9254932403564453, 0.8711993098258972, 1.4984537363052368, 1.311401605606079, 0.41670119762420654, 0.8583415150642395, -2.0013513565063477, -0.6461427807807922, 0.00589443976059556, -0.40256187319755554, -0.8454744815826416, -2.627201795578003, 1.9296289682388306, 0.0004023672954645008, -1.2162524461746216, 1.0618900060653687, -0.7672111988067627, 2.0451231002807617, -2.260556221008301, -0.19924211502075195, -1.3938945531845093, 0.211989164352417, 1.874069094657898, -0.8203643560409546, -0.0382269024848938, 1.2246893644332886, 0.018738990649580956, -0.012114285491406918, -0.4358404576778412, -0.010249640792608261, -0.7592236995697021, 0.7992653846740723, -0.5749174952507019, 0.3821181058883667, 0.2725287675857544, -0.02279457077383995, 0.2602168321609497, 0.8754417896270752, -0.016895566135644913, 1.3853181600570679, 1.5710787773132324, -0.8026222586631775, -1.0434341430664062, -0.3039321005344391, -0.46326944231987, 1.350358247756958, -1.3884649276733398, -0.8949071764945984, 0.6069228053092957, -0.6390014290809631, -0.345731258392334, -1.1723653078079224, -1.2651528120040894, 0.7159327864646912, -0.8617297410964966, -0.8614466786384583, 0.911225438117981, 0.2789554297924042, -0.14403244853019714, -1.1372623443603516, -1.3227328062057495, -0.19227297604084015, -1.5393507480621338, -0.14180900156497955, 1.2262628078460693, 0.2790403962135315, 0.2031538337469101, -1.0027111768722534, -0.14527487754821777, 0.30679839849472046, -1.432714819908142, 0.897739052772522, -1.0215762853622437, 0.03392666578292847, -0.6902816295623779, 1.8124850988388062, 0.6457597017288208, -1.5426859855651855, -1.3520127534866333, 1.6834603548049927, -0.9385566711425781, -0.8972668647766113, -0.20843468606472015, 2.0880210399627686, 0.6603980660438538, -0.006909775547683239, 0.6143787503242493, -0.18497015535831451, 0.8172205686569214, 0.8349901437759399, 1.1876310110092163, -1.3223320245742798, -0.6626423001289368], [0.30605727434158325, 0.7805796265602112, -0.5762644410133362, 0.9053855538368225, 0.6898064613342285, 1.1118930578231812, 0.9560966491699219, 0.023923195898532867, 0.749177873134613, 0.7723018527030945, -1.633603811264038, -1.4483963251113892, -1.5785495042800903, 0.5168479084968567, -0.4967825710773468, -0.8275672197341919, -1.023154854774475, 0.9673938155174255, -0.45184245705604553, 1.94573974609375, -0.40027713775634766, 0.57627934217453, 0.6822408437728882, 0.787309467792511, 1.2935668230056763, -0.8878408670425415, -0.6746721267700195, 0.2092379778623581, -0.12611617147922516, -0.7898651957511902, -1.272038221359253, -1.154111385345459, 0.35974758863449097, -0.3923361897468567, -0.7678265571594238, 0.9680098295211792, 0.46566537022590637, 1.3424370288848877, 0.12066590785980225, -1.0282782316207886, -0.23603320121765137, 0.4873284101486206, 0.28975096344947815, -1.3948094844818115, 0.9038524627685547, 0.44081029295921326, -0.7818917632102966, 1.4317923784255981, 0.1520344316959381, -2.038388729095459, -1.7443554401397705, -0.8890365362167358, 1.1599773168563843, -0.567842960357666, 0.9285789132118225, 1.4698901176452637, -1.766406536102295, 1.0223535299301147, -0.2965841293334961, -0.8975422978401184, 1.5368256568908691, 1.4003098011016846, -1.1807489395141602, -0.05319034680724144, 0.14308282732963562, 0.5500372052192688, -0.10911289602518082, -1.1609017848968506, -1.0586875677108765, -1.76366126537323, 0.08274933695793152, -2.3349416255950928, -0.249772846698761, -0.2318110167980194, 1.1279317140579224, 1.8277134895324707, -0.7305817604064941, 0.7079426050186157, -1.7616372108459473, -0.7438029646873474, 1.3686730861663818, -0.32507818937301636, -0.7709650993347168, -0.4703430235385895, 0.09025254845619202, 1.2789108753204346, 1.7081321477890015, 1.5486801862716675, 0.33983713388442993, 1.5340734720230103, 1.6609529256820679, -0.11761835217475891, 2.246685028076172, -0.07135418057441711, -0.8864744901657104, 2.2801976203918457, 1.5737059116363525, -2.137979507446289, 0.27830469608306885, 0.012455534189939499, 0.7386022210121155, -1.1112565994262695, -1.6606045961380005, -1.5095300674438477, 1.4867032766342163, 1.711819052696228, 0.5964983105659485, -0.8228355050086975, 0.5543556213378906, 0.2631390690803528, 0.21476666629314423, -0.788949191570282, -1.3118984699249268, -0.014826035127043724, 0.8112782835960388, 0.4681859314441681, -0.4941217601299286, -1.4772461652755737, -0.21608449518680573, -0.2947307527065277, 1.9123972654342651, 0.7903849482536316, -1.7483268976211548, -0.33621323108673096, -1.0296459197998047, 0.9140939712524414, -0.5667724013328552, -0.6238391995429993], [-0.28838443756103516, 1.229504942893982, 0.5171851515769958, -0.3425111770629883, -1.0916231870651245, -2.1754658222198486, 1.6473733186721802, 1.6628987789154053, -1.5112472772598267, -0.04371802881360054, 0.8869367837905884, 0.6296676397323608, -0.5113797783851624, -1.8524844646453857, -1.1645305156707764, 0.850124180316925, -0.18656465411186218, 0.7815761566162109, -1.5008487701416016, -0.1725071221590042, -0.5408879518508911, 0.17716485261917114, 0.11273666471242905, 0.4964173138141632, 0.07581649720668793, 2.0259571075439453, 0.758577823638916, 1.8644165992736816, -0.6116676330566406, 0.5548753142356873, 1.57390558719635, 0.6647818088531494, 0.8747321367263794, -0.3074638247489929, 0.05422951653599739, -0.9689679145812988, 1.008379340171814, 0.36940574645996094, 1.1206845045089722, -0.8768377900123596, -0.9008892774581909, 1.833605408668518, 0.44058194756507874, -0.15545055270195007, 1.2829148769378662, -1.1920866966247559, -0.5475942492485046, 0.23267191648483276, -0.35777634382247925, -0.3388253450393677, 0.849951982498169, 0.13151706755161285, 0.5241646766662598, 0.15361207723617554, 0.8555894494056702, 0.9088441133499146, -1.4814547300338745, -0.05494379624724388, 2.0448925495147705, -0.22995685040950775, 0.6095453500747681, -0.18966801464557648, -0.5160890817642212, 1.5451387166976929, 0.7977418899536133, -0.8996163606643677, -1.817826509475708, 0.7453457713127136, 0.9599490165710449, 0.5034705996513367, 0.2690267264842987, -0.17129702866077423, -0.7685145735740662, -0.8536059856414795, 0.251598984003067, -0.8873392343521118, -0.4415475130081177, -0.45370030403137207, 0.7965551018714905, 0.8141000270843506, -0.35147494077682495, -1.1994754076004028, -0.3949178457260132, -0.3215165436267853, 0.3266543447971344, 0.4249249994754791, -0.5484983325004578, -0.4020250141620636, 0.18373584747314453, -0.5228345394134521, 1.8765935897827148, -0.04458191990852356, 0.2814258933067322, -0.21456025540828705, -0.9685174226760864, -1.5531715154647827, -0.48738646507263184, 0.6101148724555969, 0.5405822992324829, 0.48090308904647827, 1.055571436882019, 0.24842219054698944, 1.1303478479385376, 0.736810564994812, -3.0928468704223633, -0.24174128472805023, -0.8659563064575195, -0.20934781432151794, -1.3702598810195923, 0.6585789322853088, 0.9507747888565063, -0.05988868325948715, -0.6564599871635437, 0.03135860338807106, 0.13738608360290527, -0.709484338760376, 0.2253796011209488, 0.8895920515060425, 0.29035812616348267, 0.18040165305137634, -0.5516225099563599, 0.08212706446647644, 2.2224576473236084, -0.3264574110507965, 0.7660802006721497, 0.029793277382850647, -0.8975294828414917, -2.3451290130615234], [-1.3499839305877686, 0.06062750890851021, -0.3219813108444214, 1.3383525609970093, 1.1430811882019043, -0.9785584807395935, -0.34183114767074585, 0.4132314622402191, 0.129116490483284, -2.026334524154663, 0.46033158898353577, 0.4593147039413452, 0.2684996426105499, 0.12683536112308502, -0.25417041778564453, -0.22995004057884216, 0.11163143068552017, 0.8777586817741394, 1.0489354133605957, -1.8943207263946533, 0.21465688943862915, -0.43817684054374695, 0.391426146030426, 0.6977729201316833, -0.9900363087654114, -0.13787828385829926, 0.9370670318603516, 1.5371960401535034, 0.6551438570022583, -0.07799139618873596, -0.09921342879533768, 0.26770853996276855, -0.5757303237915039, -0.4170158803462982, 0.23133555054664612, 0.6343455910682678, 0.14752477407455444, -0.04553627222776413, 0.5424863696098328, -1.2578184604644775, -0.12603537738323212, 0.25728946924209595, 0.9221214652061462, 0.3164994716644287, -1.1247317790985107, -1.39395010471344, -0.4134847819805145, 0.6580840349197388, -0.6690858006477356, -0.5816274285316467, 1.08963143825531, -0.9148760437965393, -0.15073904395103455, -0.22210177779197693, 0.2525902986526489, 0.0920630395412445, -1.2803846597671509, 0.4631401598453522, 0.08583683520555496, -1.1636006832122803, -0.6189832091331482, -1.2446705102920532, 1.6091760396957397, -0.7957372069358826, -0.5266205668449402, -0.2138223648071289, 0.48252204060554504, 0.9040921926498413, -0.6115872263908386, 0.04717395454645157, 0.10242775082588196, -0.011938015930354595, -0.17081256210803986, 0.7493060231208801, -2.9300637245178223, -0.5647062659263611, 1.5570248365402222, -2.3364417552948, -0.8715645670890808, -1.0353467464447021, -0.4096960127353668, 0.2835606336593628, 0.6829306483268738, 0.7026405334472656, 1.2410979270935059, -0.060086678713560104, 0.798767626285553, 1.1353378295898438, 1.1965785026550293, -0.16099271178245544, -2.0032665729522705, 0.2545342445373535, -0.95892333984375, 0.5541643500328064, -0.8521580100059509, 0.9454776048660278, 0.6307939887046814, -0.4652908742427826, -0.19315126538276672, 1.4110150337219238, 0.6856887340545654, -0.4185973107814789, -0.37123847007751465, -1.0416951179504395, -0.5673456788063049, -0.6250895261764526, 0.06242845952510834, 0.9493932723999023, 1.6285693645477295, 0.24276487529277802, -1.0353662967681885, -1.0187203884124756, 0.5531993508338928, 1.128753423690796, -0.29771628975868225, -0.7961167097091675, -0.744092583656311, -0.07826157659292221, 1.2638319730758667, -0.8122905492782593, -0.16019250452518463, -1.3095625638961792, 0.23010636866092682, -1.113036036491394, 0.5164093375205994, 1.135655403137207, 0.3679198622703552, -0.6410908102989197], [-0.3407280743122101, -0.3079412579536438, 0.3907443583011627, -0.6599884629249573, 1.2232311964035034, -0.16381549835205078, -0.45991188287734985, 0.0629974976181984, 0.8390544652938843, 1.1515673398971558, -0.46643972396850586, -0.3354640603065491, -0.5288702845573425, -0.6251317262649536, 0.9319895505905151, 0.8613195419311523, -2.0343258380889893, 0.79665607213974, -0.6527785062789917, 0.12571747601032257, -0.1814848631620407, -0.11063793301582336, 1.6140271425247192, 1.040432333946228, 1.257324457168579, -0.6302692294120789, -0.9156805872917175, 0.7687913179397583, 0.6032195687294006, -2.263765573501587, 0.6313089728355408, -1.3713654279708862, 0.07773379981517792, 0.6826238632202148, 0.1889321208000183, 0.79400634765625, 0.6795505881309509, -0.7338675260543823, -0.3766983449459076, 1.5377005338668823, 0.928898274898529, 0.7170214653015137, -1.4944186210632324, 0.71625155210495, 0.22482797503471375, 0.45376312732696533, 0.07134635746479034, -1.767893671989441, -1.260441780090332, -0.5003819465637207, 0.6094832420349121, -1.0835850238800049, -0.02360244281589985, 1.1314935684204102, 0.9463996291160583, 0.13995200395584106, 0.16100390255451202, -0.11303944885730743, 1.167875051498413, -0.43377912044525146, 0.234737828373909, -0.27475038170814514, -0.5139224529266357, 1.9010717868804932, -1.437422275543213, 0.11990046501159668, -0.8753716945648193, -1.2305419445037842, -1.3801349401474, 0.5373098850250244, -2.1335222721099854, 1.3358761072158813, 0.2721637785434723, -1.2694716453552246, 0.5529839992523193, 0.33918875455856323, 1.435856580734253, 1.4046992063522339, 1.175610899925232, 0.0844373032450676, 1.3487709760665894, 0.052460357546806335, -0.30999210476875305, -0.19201494753360748, 0.3621770143508911, 1.0983225107192993, 1.150497555732727, -1.6954927444458008, -2.2177507877349854, -0.9191922545433044, -0.009650071151554585, -0.6892582774162292, 1.5594483613967896, -0.6694242358207703, 1.5956991910934448, -2.0631513595581055, -1.0208760499954224, 0.3566441833972931, 0.7024881839752197, -0.34844449162483215, 0.9445003271102905, -1.2833163738250732, -0.6985456347465515, -0.10994455218315125, -0.7679070234298706, 1.486873984336853, -0.8883533477783203, 0.8756511211395264, -0.7162376642227173, 0.7560921311378479, 0.408598393201828, 0.6384705305099487, 0.17708081007003784, -2.5340497493743896, -0.7454557418823242, 0.7014496922492981, 1.1418957710266113, -0.08573832362890244, -0.3541591763496399, -1.1109603643417358, -1.439188003540039, -0.15973062813282013, -0.7309547662734985, -0.4110972583293915, -1.46150803565979, 0.1307549625635147, 0.5794295072555542, 0.6787751913070679], [-2.625915765762329, -1.1935524940490723, 0.5748487710952759, -0.3746599853038788, -0.675801157951355, -0.2498251497745514, -2.633260726928711, 1.4793974161148071, 0.7632115483283997, -1.3059399127960205, -0.14218296110630035, -0.8820458650588989, 0.36323440074920654, 0.4382186233997345, -0.6301828026771545, 0.8867761492729187, -1.4442570209503174, -0.5289490222930908, 0.5852635502815247, -1.8791794776916504, 0.4434504806995392, 1.0505753755569458, 0.7789092659950256, 0.16314220428466797, 2.7528536319732666, 0.5041229724884033, -0.3869071900844574, 0.5409283638000488, -0.592961311340332, -1.8853487968444824, -0.9773826599121094, -1.4412349462509155, 1.964673638343811, 0.7472817301750183, -0.28003254532814026, -0.1807965636253357, 2.1950571537017822, 0.560342013835907, -0.7517966032028198, -0.5301626920700073, -0.13342787325382233, -0.07718723267316818, 0.32631441950798035, 1.6074753999710083, -0.9367629885673523, -0.6767174601554871, 1.1965974569320679, 0.7920621037483215, -0.9186468124389648, 0.497546911239624, -0.915591299533844, -0.6218332648277283, -0.8286352753639221, 1.4714462757110596, 2.0959975719451904, -1.256091833114624, -0.9561044573783875, -2.4877371788024902, -0.4448241889476776, -0.8743978142738342, 0.691389799118042, 1.8945367336273193, 0.32484155893325806, 1.4972914457321167, 0.6437262296676636, 1.260390281677246, 2.350904703140259, -0.04809819534420967, 0.0812675952911377, 0.33761730790138245, 0.20024819672107697, 2.475346565246582, -0.6496871709823608, 0.7101728320121765, 1.653613805770874, 1.0444555282592773, -0.6848512887954712, 0.17924724519252777, 0.4814583957195282, -1.0006994009017944, 0.8309683799743652, -0.6392596960067749, -1.2369798421859741, -1.5519773960113525, 0.3708873987197876, -0.2809249758720398, -0.08213234692811966, 0.07023351639509201, -0.8921083211898804, 0.354800283908844, 0.052087508141994476, 2.347444772720337, 0.7537457346916199, -0.8622528910636902, -0.7624430060386658, 0.5667943954467773, 1.7403346300125122, 0.9797849059104919, 1.365064263343811, -0.9807199835777283, -0.4089040756225586, 0.3936479985713959, 1.6619925498962402, -0.6028218269348145, -0.10581842064857483, -0.8130231499671936, -0.7491234540939331, -0.07783469557762146, -0.849422037601471, -0.3052648901939392, -0.1756647229194641, -0.2105366736650467, 0.06418871879577637, 0.5225225687026978, 0.9621354937553406, 1.4337291717529297, -0.22147929668426514, 0.7500576376914978, -0.8710131645202637, 0.87315434217453, -0.49393653869628906, -0.5393940210342407, -1.6711318492889404, -0.849118173122406, 0.017742980271577835, -0.4696486294269562, 0.45195308327674866, -0.5333676934242249], [-0.07967420667409897, -2.109743595123291, -1.3277074098587036, -0.4604146480560303, 0.09054752439260483, -0.9151083827018738, -0.7620640397071838, 1.2757669687271118, -1.8830862045288086, -1.4414798021316528, 0.7427312731742859, 0.7059305310249329, -0.6562780737876892, 0.4392721951007843, 1.034710168838501, -1.0274097919464111, -0.5055595636367798, 0.5935952663421631, 1.5594464540481567, 0.06643616408109665, -0.9494897723197937, 0.05760333314538002, 0.47125664353370667, -0.12397126108407974, -1.3699417114257812, -0.31037136912345886, -0.9797860383987427, 1.0346486568450928, 1.4449267387390137, -0.4703214168548584, -0.5063058733940125, 0.811802327632904, -0.3540983498096466, 0.8964121341705322, 2.255685806274414, 0.15913026034832, -1.9806119203567505, 0.08407635986804962, -0.02730083093047142, -0.05144728720188141, 0.19079537689685822, 1.89110267162323, 0.4896867275238037, 0.40885549783706665, 0.3656867742538452, 0.36017847061157227, -1.1757837533950806, 0.3712160885334015, -0.37810447812080383, -1.4421483278274536, -0.4159139394760132, -1.8705990314483643, -0.051247674971818924, 0.5665778517723083, -0.5642237663269043, 2.897442579269409, 0.3586767911911011, 0.5347760915756226, 1.485940933227539, -0.3993053734302521, -0.581494152545929, -0.3031138479709625, 0.8932608366012573, 0.46228328347206116, -0.8110837936401367, 0.6672790050506592, -1.3478633165359497, 0.6067157983779907, 1.225077509880066, 0.41592589020729065, -0.5081217885017395, -0.5104361176490784, 0.845140814781189, 0.5770863890647888, -0.19935496151447296, 0.11900340765714645, -0.48481184244155884, -0.5088120102882385, -2.067775249481201, 0.44079023599624634, 0.2998671233654022, 0.2904575765132904, -0.7158374786376953, 0.26750659942626953, 1.3245660066604614, -1.2837944030761719, -0.8491933941841125, -0.8649922609329224, -0.07973187416791916, -0.9029890298843384, 0.6352540850639343, -0.22411026060581207, -0.21367652714252472, -0.03358088806271553, 0.26723888516426086, 0.9747563600540161, -0.8582379221916199, -1.6518343687057495, 1.5035018920898438, -0.2413250207901001, -0.1792101413011551, -0.2709765136241913, -0.07494820654392242, -0.9353859424591064, -1.1876858472824097, 0.7677134871482849, 0.07747901976108551, -0.3139905333518982, 1.2564581632614136, 1.1159981489181519, 0.5745071172714233, 0.0017274252604693174, -2.002943992614746, 0.8486572504043579, 1.3025579452514648, 0.7591692805290222, -2.756544828414917, -0.6997043490409851, 0.4729354977607727, 0.8161385655403137, -0.210682675242424, -1.0704617500305176, 1.245139241218567, 0.19773037731647491, -1.3668967485427856, -1.0795021057128906, 0.1745511144399643, -0.055112630128860474], [1.717882752418518, -0.9318499565124512, -0.9199734926223755, -0.07535926252603531, -0.2405010461807251, -0.13259191811084747, -0.37574484944343567, 0.3480541408061981, 0.40067803859710693, 0.06598590314388275, -0.2569146752357483, 0.20696714520454407, 1.3574628829956055, 0.8053176999092102, -1.7179279327392578, -0.49411699175834656, 0.550873875617981, 0.7724967002868652, 1.8653181791305542, -1.697320818901062, 2.579528570175171, 0.02493946999311447, -0.24030916392803192, 1.0044294595718384, 0.6725549697875977, -0.818178117275238, -0.14395993947982788, 0.710218071937561, -0.06518065184354782, -0.056428276002407074, 0.317402720451355, 0.9333640933036804, -0.5511026382446289, -0.8212307095527649, 0.6318318843841553, 0.4613930881023407, 0.38998693227767944, 0.46101441979408264, 0.9682820439338684, -0.6468054056167603, -0.8175780773162842, 0.7559724450111389, 0.8091515302658081, -0.15653133392333984, -1.8047385215759277, 0.3067604899406433, 0.8709015250205994, -1.009616732597351, 0.6547167897224426, -1.9760457277297974, 0.4021969139575958, 1.07175612449646, 1.9497607946395874, 1.8920643329620361, 0.2105308175086975, -0.26367637515068054, -0.8674807548522949, -0.38732394576072693, -0.5240328907966614, -1.574899435043335, -0.32203516364097595, 0.1175883412361145, -1.1957274675369263, 0.1065526232123375, 2.0370097160339355, 0.086405448615551, 1.2933838367462158, 1.6774152517318726, 0.9830180406570435, -0.027062885463237762, 0.7350164651870728, -1.0258151292800903, -1.5319827795028687, 1.7225896120071411, 1.092753291130066, -0.12908783555030823, -0.4239305853843689, 0.033127158880233765, 0.07020560652017593, -0.8427806496620178, 0.14255905151367188, -0.9563677906990051, 1.2131234407424927, 1.9115253686904907, 0.8352987170219421, -0.7886391878128052, 0.2979409694671631, -2.255265951156616, 1.018640160560608, 1.717935562133789, 0.007607111241668463, 0.28224825859069824, -1.3928309679031372, -1.2066535949707031, 0.7069695591926575, -0.912824273109436, 0.3481687307357788, -0.869526207447052, -1.4989831447601318, 0.5367974042892456, -0.19121825695037842, 0.16420194506645203, -2.6219539642333984, -1.2408348321914673, -0.19329112768173218, -1.5780730247497559, -1.2929788827896118, -0.5659037232398987, -0.008245224133133888, 2.280320167541504, 0.8324252367019653, 0.7103879451751709, -1.009933352470398, -1.1635541915893555, 0.3345074951648712, 0.6534022092819214, -0.8074853420257568, -1.0978403091430664, -0.7666744589805603, -0.2318626046180725, -0.09359872341156006, -0.809359610080719, -0.5176282525062561, 0.47109004855155945, 0.5794051885604858, 0.9284871220588684, 0.8512849807739258, 0.17897453904151917], [0.6994020938873291, 0.5937939882278442, -1.3514509201049805, -0.2973220646381378, -0.43040648102760315, -0.07940945774316788, 0.2189427614212036, 0.9215789437294006, 0.9453113675117493, 0.7006887197494507, -0.6620871424674988, -1.0314397811889648, 0.6808241605758667, -0.6787015199661255, 0.4804411232471466, -0.03414566069841385, -0.24848449230194092, 0.7231177091598511, 0.6766975522041321, 1.5451170206069946, -1.3542239665985107, 0.8850880861282349, -1.226577639579773, 0.7610995769500732, -0.14312970638275146, -0.14693476259708405, -0.06321121007204056, 0.01053378451615572, -0.2582240700721741, -1.8963673114776611, 0.7052164077758789, -0.7876380681991577, 0.42160528898239136, 1.9877680540084839, -1.094022512435913, -0.6303920149803162, 0.7687181830406189, -1.0768027305603027, -0.5082413554191589, -2.0456442832946777, -0.584266722202301, -1.2113103866577148, -0.8853459358215332, 0.6914160847663879, -1.1134300231933594, 0.04098283872008324, -0.37502622604370117, 1.2892727851867676, -0.5964002013206482, 0.04262232407927513, -0.47990357875823975, 0.2071327120065689, -1.0546557903289795, -0.6366490125656128, -0.048828545957803726, 0.4932273030281067, 0.9045617580413818, 0.47323936223983765, 3.4000980854034424, -0.45886486768722534, -0.04254930466413498, 0.535060465335846, 0.5208101272583008, 0.5251160264015198, 1.247256875038147, 1.2555363178253174, -1.2909544706344604, 0.5020138621330261, -1.1248146295547485, -0.5554494261741638, 1.1226774454116821, -0.8617632389068604, -0.8962042927742004, -0.7620439529418945, 0.6432371735572815, -0.7146930694580078, -0.737044632434845, 1.659058928489685, -0.9026188254356384, -0.45132550597190857, 1.5655964612960815, -1.7570319175720215, -0.9718447327613831, 1.3618324995040894, 0.41701361536979675, -2.605494976043701, -0.9109000563621521, 0.40472689270973206, -0.47540342807769775, 0.274722695350647, -0.570297122001648, -1.0885485410690308, 0.5644838809967041, -0.8018166422843933, 0.42988160252571106, 1.462821364402771, -2.4559478759765625, -2.2528505325317383, -0.17121505737304688, -0.6258945465087891, 1.9263641834259033, -0.5107753872871399, -0.6953617334365845, 0.5988368988037109, 0.06689313799142838, 1.299134612083435, 0.8709858655929565, -1.053917407989502, -1.4657683372497559, 0.44618409872055054, -0.93891841173172, 0.07642126083374023, 0.8090226054191589, 0.15264232456684113, 0.09031268209218979, 0.46702244877815247, -1.2430927753448486, -0.9188628196716309, -0.953346312046051, -1.6752721071243286, 0.35005244612693787, -0.33026590943336487, -0.16615664958953857, 0.8602778911590576, -2.7890403270721436, -0.8748360276222229, 1.3335163593292236, -0.8800442218780518], [-0.4469144642353058, 1.000727891921997, 0.08131477981805801, 0.05968043953180313, 2.3211898803710938, 0.06038729473948479, 1.7896249294281006, 1.115171194076538, 0.5857945084571838, 1.5714876651763916, -1.5191149711608887, 1.1752562522888184, -1.4523147344589233, -0.04677651450037956, -1.5163074731826782, 0.4986720383167267, 0.46383312344551086, 1.8473907709121704, 0.397869348526001, -0.6641848087310791, -1.6883164644241333, -0.8764596581459045, -1.6669001579284668, 1.6135187149047852, -0.02666744403541088, 1.726351261138916, 1.100170373916626, -0.11659368127584457, -0.20120380818843842, 0.4340260624885559, 0.1718139946460724, 1.4435471296310425, 0.18339036405086517, 1.0927722454071045, 1.8023244142532349, 1.4842793941497803, 1.4841892719268799, -1.6868553161621094, -0.18268385529518127, 0.2748543322086334, 1.5916060209274292, 0.9881958961486816, 0.034516479820013046, 0.3210408389568329, -0.6512587666511536, -0.670412003993988, -0.7896447777748108, -0.6672936677932739, 2.0057830810546875, 1.6097731590270996, 0.3425661623477936, 0.16030442714691162, -0.5131291151046753, 0.6804363131523132, -0.6458422541618347, -0.6593215465545654, -1.3470771312713623, 0.7412048578262329, -0.7820204496383667, 1.0243825912475586, -0.48364198207855225, -0.3604968786239624, 1.2991136312484741, 0.17136800289154053, 0.20314843952655792, 0.8004807829856873, -0.39217692613601685, 0.16086030006408691, 0.026522696018218994, 0.03652278333902359, -0.17103591561317444, -0.051535043865442276, 0.32150182127952576, -0.8651715517044067, -1.689283013343811, -0.8132241368293762, 0.7035687565803528, -0.6465286016464233, -1.9294100999832153, 0.12837818264961243, 1.2423149347305298, -2.0987389087677, -0.6112635135650635, 1.9383578300476074, -1.4755545854568481, -1.2421271800994873, 2.428337574005127, -0.09226848185062408, -1.4411873817443848, 0.17837317287921906, 0.9382153153419495, -1.5903048515319824, -0.05708297714591026, 1.787678599357605, 0.6746013760566711, -0.45495662093162537, 0.3750718832015991, 1.7381906509399414, -1.3245917558670044, 1.4414464235305786, 1.0225163698196411, 0.7969621419906616, 0.5724827647209167, 0.5353431701660156, -2.1764416694641113, -0.554124116897583, -0.6102178692817688, 1.089172124862671, 0.3025391697883606, 0.3867137134075165, 0.5238304138183594, 0.7948360443115234, 0.3084723651409149, 0.15559174120426178, 1.0974966287612915, 1.0299856662750244, 0.35855916142463684, 1.1325589418411255, 1.0137747526168823, -0.5106443166732788, -1.0077975988388062, -0.5712906122207642, 0.10489212721586227, -0.30565401911735535, -1.0956618785858154, 0.392406165599823, 1.188585877418518, 1.0984525680541992], [1.3615931272506714, 0.8669774532318115, 0.7991576790809631, -1.2112915515899658, -0.04705790802836418, -1.234076976776123, 1.6337519884109497, -0.4587683081626892, -0.9608466625213623, 0.7984347939491272, 0.7483689785003662, -0.5862395167350769, 0.5087453126907349, -0.3815731406211853, 2.0689737796783447, -1.6942059993743896, -0.9769577980041504, 1.0384624004364014, 0.8802613019943237, -0.1493622362613678, 0.9652931690216064, 0.028567718341946602, 1.6351697444915771, -0.3723636865615845, 2.4500985145568848, -0.4040619432926178, -0.7122130393981934, 0.7381603717803955, 0.057744018733501434, 0.5024459362030029, 0.7036677002906799, 0.9459052681922913, -1.6834429502487183, 0.3482443690299988, -0.9686744213104248, 0.7082986831665039, 0.028638839721679688, 0.26646900177001953, -0.0192844420671463, 1.0040550231933594, 0.5631778836250305, 0.0858723595738411, -0.054756924510002136, 1.2486916780471802, 3.002342462539673, -0.923733115196228, -0.3985023498535156, 1.4361165761947632, -0.5491176247596741, -0.6135327816009521, 0.4947872459888458, -0.9783979654312134, -0.852230966091156, 0.42476576566696167, -1.308567762374878, -1.32681405544281, 1.525182843208313, 0.19191129505634308, 0.6885896325111389, 0.14313474297523499, -0.9684855937957764, 0.33034858107566833, 1.6399650573730469, -0.783636748790741, -1.5315721035003662, -1.5093485116958618, -0.14814670383930206, 1.4978156089782715, 0.15128888189792633, 0.7141242623329163, -1.2200651168823242, -0.6885856986045837, -1.1594797372817993, -0.19136066734790802, 0.10765586048364639, -0.7311707139015198, 2.0676608085632324, -0.9363139271736145, 1.118293046951294, 1.7136963605880737, -1.1344449520111084, 2.041957139968872, 1.2475316524505615, -0.3996990919113159, 0.7678382396697998, -0.6794684529304504, 0.5157046318054199, -0.24603013694286346, 0.9859802722930908, 0.08339427411556244, -0.045368071645498276, 0.12339940667152405, 0.4561033546924591, -0.20432767271995544, -2.699751138687134, 0.8260254263877869, -0.35508888959884644, 0.6805624961853027, 0.41274720430374146, 0.7492559552192688, -0.6114166975021362, 1.005536675453186, 0.47609710693359375, 1.0466947555541992, 0.5204699039459229, 0.47267207503318787, -0.7015483379364014, 0.16302815079689026, -3.031088352203369, 1.3788878917694092, -0.3671608567237854, 1.8174378871917725, -0.1042625829577446, 1.0941245555877686, -0.5944243669509888, -1.1985504627227783, -0.3623485565185547, 1.6425305604934692, -0.9025560617446899, -2.265734910964966, 1.7907030582427979, 0.6264730095863342, 0.31168317794799805, 0.3415835499763489, -1.6316750049591064, -0.6966519951820374, -0.46264374256134033, 0.8113325834274292], [-0.7616905570030212, -0.1472635269165039, -1.9798429012298584, 0.9278824925422668, -1.4816519021987915, 1.6365540027618408, -0.16957029700279236, 0.23869021236896515, 1.5750863552093506, 0.9888615608215332, -1.679919719696045, -0.0199175663292408, 1.433064579963684, -0.4184965491294861, -0.7837279438972473, -0.20794853568077087, -0.8562656044960022, -0.7659407258033752, 0.22080321609973907, 0.637130856513977, -1.193640112876892, 0.591628909111023, 0.9109915494918823, 0.09897714853286743, 0.6827710270881653, 0.8102020621299744, -0.6294366717338562, -0.03428465500473976, -0.1754053682088852, 0.2912119925022125, -0.17252607643604279, -0.594326376914978, -1.4347079992294312, -0.008658280596137047, -1.7570245265960693, -0.4098082482814789, -0.12471843510866165, -0.6132400631904602, -0.35746005177497864, 0.5472580194473267, 1.4728163480758667, 0.4515949785709381, -0.9966200590133667, 1.1708340644836426, 0.3919357359409332, -0.3648669123649597, -0.4394185245037079, -0.1360563039779663, 0.16925457119941711, 0.8205835223197937, 1.128564715385437, -1.255501627922058, 0.6922328472137451, -0.30974704027175903, -1.5091509819030762, 0.5397696495056152, -1.0769466161727905, -0.020918667316436768, 1.2552061080932617, 1.8878586292266846, 0.25979137420654297, 0.12304309755563736, -0.14735841751098633, -0.6023425459861755, -0.3140285611152649, 0.2938038110733032, 0.24365274608135223, -0.11879279464483261, -0.2166012078523636, -0.24259334802627563, 0.28161314129829407, 1.1917800903320312, -1.7205853462219238, -0.8792905807495117, -0.6018775701522827, -0.45633038878440857, 1.6391876935958862, -0.42586061358451843, -0.2828163206577301, 3.129873752593994, -1.0636619329452515, -0.19620586931705475, -0.5760197043418884, -1.3199502229690552, -0.6254155039787292, 0.5968660116195679, 1.2479255199432373, 0.15276218950748444, 0.8378325700759888, -0.5135993361473083, 0.1514149010181427, -0.4068385660648346, -1.379028081893921, 0.18777120113372803, 1.3630552291870117, 0.45070621371269226, 0.4265548288822174, 1.1731681823730469, 0.5286651253700256, -0.035651665180921555, 0.946568489074707, 1.7727556228637695, 1.5304913520812988, 0.11873123794794083, -2.0215187072753906, 0.8345410823822021, -0.8745594620704651, -0.6946808695793152, -0.7623348832130432, 1.2036094665527344, -0.2351565957069397, -0.8544948101043701, 1.9033949375152588, 0.0354611873626709, -0.7574124336242676, -0.8532359004020691, 0.518759548664093, -0.6250362992286682, 0.7991009950637817, 0.42186808586120605, -1.2880386114120483, -0.04323114454746246, 0.01030031405389309, -1.9575167894363403, 0.2993778884410858, 1.9470949172973633, -1.4301625490188599, 1.429895281791687], [-0.40304678678512573, 0.04719771072268486, 0.44587090611457825, 2.323974847793579, 0.3294289708137512, 0.8277478218078613, -0.05852990597486496, 1.1226316690444946, 0.40380772948265076, -0.7932400107383728, -0.46170827746391296, 1.7773520946502686, -0.025660935789346695, 0.21764835715293884, 0.7126398682594299, -0.04630277678370476, 1.3923391103744507, 0.39195436239242554, -0.6036860346794128, -0.01952611282467842, 0.6470229029655457, 0.18000881373882294, -1.3446656465530396, -2.0622951984405518, 1.4366295337677002, 1.238539695739746, 1.3421047925949097, 2.1538209915161133, -0.25371241569519043, -0.6666059494018555, 0.16450218856334686, -0.0960908830165863, 1.804192304611206, 0.3329637944698334, 0.5147827863693237, 0.37653210759162903, 0.472869873046875, -0.5151407718658447, -0.5209505558013916, 0.05387145280838013, -0.027817782014608383, 0.33361876010894775, -0.40234503149986267, 0.2387198954820633, -0.5030383467674255, -1.8359342813491821, -0.36076903343200684, -0.08087658882141113, 1.3900736570358276, -0.4313397705554962, -1.6098555326461792, -1.5007500648498535, 0.34332212805747986, -0.7462628483772278, -0.8320144414901733, -0.5163279175758362, -0.30625420808792114, -1.1556899547576904, -1.663832664489746, 0.3223992884159088, 0.06830602884292603, -0.8244823813438416, 0.563687801361084, -0.6722869873046875, -0.752914547920227, 0.4358616769313812, -0.059536851942539215, 0.32377883791923523, -0.46079742908477783, 1.1906940937042236, -0.190063938498497, 0.03166862577199936, -0.9272366166114807, 0.03995983302593231, -1.2361425161361694, 0.3322316110134125, 0.666290819644928, 0.705091118812561, 1.6006020307540894, 1.2859079837799072, -1.5531690120697021, -1.0251591205596924, 0.10325460880994797, -0.5563613772392273, -1.3508397340774536, 0.32489246129989624, -1.437947154045105, 0.1371431201696396, -0.07744815945625305, 0.6778496503829956, -0.4774189889431, 0.21884270012378693, -0.8007838726043701, -1.6546850204467773, -0.1130615845322609, 0.04459838941693306, -0.4444480240345001, -0.05256856977939606, 1.436376929283142, -0.5273035764694214, 1.387003779411316, 0.4233933389186859, 0.9594221115112305, 1.9103609323501587, 0.5725116729736328, 2.6175646781921387, 0.09397182613611221, 1.4113237857818604, 1.2778173685073853, 1.7331676483154297, -1.285401463508606, -1.726671576499939, 0.9870120882987976, -0.6104874610900879, 0.12094791978597641, 0.43965020775794983, -1.1330231428146362, 0.4147966504096985, 0.8183626532554626, -0.5138436555862427, 0.7894700765609741, -0.7983747720718384, 0.35713833570480347, 1.7087634801864624, 1.179539680480957, -1.0881404876708984, -1.516024112701416, -0.3078314960002899], [0.7955756783485413, -0.024998711422085762, -1.2809679508209229, 0.835421621799469, 0.9936002492904663, -1.0342066287994385, -0.774443507194519, -0.6773445010185242, 0.8375656604766846, 1.518025517463684, -1.8602293729782104, -1.4363007545471191, -0.7984493970870972, 0.29711318016052246, -0.3672723174095154, 0.9406287670135498, 0.43116235733032227, -0.6113520860671997, 0.7184137105941772, -0.17478154599666595, 0.9335442185401917, -1.000120759010315, -0.5918848514556885, -0.04476618766784668, -0.3825332820415497, -0.9412626028060913, 1.7847212553024292, 0.6774380207061768, -1.86476731300354, -0.2517193853855133, 1.1960034370422363, -0.6493229866027832, -0.18657642602920532, -0.2672053575515747, -0.6680480241775513, -0.45109865069389343, -0.6825746893882751, 1.6258264780044556, -0.25636985898017883, -0.26357680559158325, -0.2963077127933502, -1.286354899406433, 0.9961406588554382, 0.34482046961784363, 2.2250304222106934, 0.5171441435813904, 0.35630422830581665, -2.578641891479492, 0.6089588403701782, -1.5352036952972412, -0.6833320260047913, -2.1156089305877686, -1.0137381553649902, -0.4647615849971771, -0.14631786942481995, -1.5687357187271118, 0.8240407705307007, 0.26248374581336975, -1.2144736051559448, 1.348787546157837, -0.9157403707504272, -1.0890779495239258, -1.3164538145065308, -0.9862922430038452, 0.7228510975837708, 0.7568955421447754, 0.06230717897415161, -0.6567539572715759, 0.1311325877904892, -0.15687133371829987, -0.1338246613740921, -1.1566603183746338, -0.471213698387146, 2.2463603019714355, 0.17334023118019104, 1.2733060121536255, -1.8950594663619995, -0.03881462663412094, 1.1324670314788818, 0.2172582894563675, 0.35165631771087646, -0.6164163947105408, -0.6208509206771851, 0.011602486483752728, -1.3361016511917114, -0.28197920322418213, 1.279272437095642, -0.24092425405979156, 0.06589486449956894, -0.526326060295105, 2.247493267059326, -1.0580452680587769, -1.412348985671997, -0.620581328868866, 1.8499677181243896, 1.383234977722168, -0.8550403714179993, -2.484227418899536, 0.7802193760871887, -0.4661449193954468, -1.6515763998031616, 1.017671823501587, -0.68281489610672, 0.7022343277931213, 0.13473553955554962, -0.7019177675247192, 0.5266043543815613, -0.7800302505493164, 0.48479151725769043, 1.1473554372787476, -0.6842763423919678, 0.18890568614006042, 2.218477725982666, 1.5028871297836304, -0.9683089852333069, -1.0823248624801636, 1.205339789390564, 0.6695118546485901, -1.0212308168411255, -0.7980400323867798, 0.2692992687225342, -0.30594971776008606, 0.7694517970085144, 0.04423872381448746, 0.025275234133005142, 0.1589653193950653, 1.6828206777572632, 0.8220945000648499]], [[0.2009749412536621, -0.2522408962249756, 2.846161365509033, 0.7693058252334595, -0.42246049642562866, -1.1862107515335083, 1.0818560123443604, 0.5657727122306824, 0.4896753132343292, 0.5148794651031494, -0.35115617513656616, 1.4170728921890259, -0.5747695565223694, 2.0288944244384766, 0.6464827060699463, 0.7480242848396301, 1.515183448791504, 1.2418891191482544, -1.1880265474319458, 2.6015625, -0.7523059248924255, 0.9635169506072998, -0.34333401918411255, 2.0430469512939453, -0.6607273817062378, 0.17714114487171173, 0.3797229528427124, -0.23435869812965393, -0.3135731518268585, 0.3647592067718506, -0.9422955513000488, 1.5605285167694092, 1.0989971160888672, -1.1866880655288696, 0.33838751912117004, 0.5668931603431702, 1.7071924209594727, 1.050128698348999, 1.0623433589935303, 0.15776029229164124, -0.34356218576431274, -1.8990380764007568, -1.0690668821334839, -0.12711018323898315, 1.0068399906158447, 0.30502283573150635, 1.6751421689987183, 0.11353346705436707, -0.3752429783344269, 0.3477734625339508, 0.08108556270599365, -0.40192827582359314, 1.5125751495361328, 0.24439682066440582, -1.7143176794052124, -0.8776612877845764, -0.09676235169172287, 1.0145925283432007, -0.3479854166507721, 1.190712332725525, 2.3226842880249023, -0.30526840686798096, 0.37486305832862854, -0.31853508949279785, 0.35942310094833374, 0.5502527356147766, 1.332313060760498, -0.6194548010826111, -0.5930128693580627, 0.6802127957344055, 0.1960754543542862, -0.7187647223472595, -0.6969466805458069, 1.7146047353744507, -1.0400179624557495, -0.7949329614639282, -0.038553230464458466, 1.1099501848220825, -1.0766558647155762, 1.7626996040344238, 1.6422107219696045, -1.306127667427063, 0.8031930923461914, 0.21470589935779572, -1.1133352518081665, -1.1548858880996704, 0.18403957784175873, 0.9270617365837097, 0.17460399866104126, 0.48664188385009766, 1.226959466934204, -1.4970895051956177, 1.0633573532104492, 0.8274205923080444, 2.1267073154449463, 0.39388811588287354, -0.5062462091445923, 0.2959040403366089, -0.6502147912979126, -1.7308423519134521, 0.11111972481012344, -2.0101122856140137, -1.4843487739562988, 1.0196402072906494, -2.0129940509796143, 1.2303463220596313, -1.7225888967514038, -0.3906906843185425, 1.7793406248092651, 0.1820356398820877, -1.2738769054412842, 0.7689806222915649, 0.6053522229194641, -1.7322156429290771, 1.0134453773498535, 0.7932992577552795, 0.8956733345985413, 0.26381370425224304, 0.07067001610994339, 2.791790246963501, -0.9509374499320984, 1.627804160118103, 0.45422595739364624, 0.30514466762542725, -0.5196285843849182, 1.3068486452102661, -1.4362330436706543, -0.4582448899745941], [0.7173287868499756, -0.21838490664958954, 0.4527679681777954, 1.178863286972046, 1.8468784093856812, -0.6322828531265259, -0.18818515539169312, 1.7455631494522095, 0.09165128320455551, -0.6396299004554749, 1.4920521974563599, -0.04057512804865837, -0.5294369459152222, 2.4980337619781494, -0.3622428774833679, 2.599586248397827, -0.42273348569869995, -0.59418123960495, -0.8070164322853088, 0.9112259745597839, -0.5739063620567322, 0.6924768686294556, 0.45251893997192383, -0.20801489055156708, -1.1109821796417236, 0.5589223504066467, -0.7979239821434021, -1.1907012462615967, 1.0319103002548218, -0.24923774600028992, 0.14593425393104553, 0.0653180181980133, -2.2027859687805176, 0.2734416425228119, 0.38130417466163635, -0.006107541732490063, -0.9867634177207947, -0.6501626372337341, -2.886587619781494, 0.4775576889514923, -0.41177886724472046, 1.1364424228668213, -1.410762906074524, 1.08281672000885, -0.6485987305641174, -0.20122447609901428, 0.9837484359741211, 0.2992566227912903, 0.8136546015739441, 0.08031371980905533, -1.1939681768417358, 0.6339618563652039, 0.8154175281524658, 0.592093825340271, 0.9512891173362732, -0.37594330310821533, -0.6029202938079834, -0.3748472034931183, 0.8526932001113892, -1.134995460510254, -0.7145019769668579, -0.6596623063087463, 1.3807947635650635, 0.9011701345443726, 0.9314730167388916, 0.44778090715408325, -0.46030569076538086, 0.44320330023765564, 0.5135899186134338, -0.8367509841918945, 1.923082947731018, -0.41668158769607544, -1.6713978052139282, -0.28788015246391296, -0.02860921621322632, -1.1101305484771729, -0.5480583310127258, 1.7321521043777466, -0.7010091543197632, 0.9912029504776001, 0.282235324382782, -2.074864625930786, 0.4869248867034912, 0.7744019031524658, 0.8179914951324463, 2.48795747756958, -1.0634931325912476, 0.7967175841331482, 0.17433132231235504, -0.4458339512348175, 0.7263875603675842, -0.3324293792247772, -0.07848919183015823, -0.17295867204666138, 1.9001023769378662, 0.7012487649917603, -0.2231179177761078, 1.561545729637146, -1.4367656707763672, -0.20046298205852509, -0.09231311827898026, 0.08488225936889648, 0.25516897439956665, -1.8285596370697021, 0.1620519608259201, -0.5566874146461487, 2.277808427810669, -0.018159521743655205, 0.6638153195381165, 1.5135010480880737, -2.045621156692505, 0.31181100010871887, 0.154279425740242, -1.6535488367080688, 0.07081645727157593, -0.5628131031990051, 0.02509542740881443, 0.40999555587768555, 2.214585304260254, 0.6049882769584656, -0.005475587677210569, -1.019637942314148, -0.06562685966491699, -0.302105575799942, 0.7837657928466797, -0.6706581115722656, -0.2161470502614975, -0.280788391828537], [-0.38678523898124695, -0.9022005796432495, -1.3982551097869873, 1.0452357530593872, -0.4263770580291748, -0.22036689519882202, -1.711543083190918, 0.0037273711059242487, -0.44860154390335083, -1.7007102966308594, -0.19354330003261566, 0.03685760498046875, 3.079911470413208, 0.552911639213562, 1.4051834344863892, 0.034019652754068375, -2.5717833042144775, 1.2073469161987305, 0.703353762626648, 0.23136745393276215, -0.45486631989479065, -0.05304686352610588, 0.4601863622665405, -0.03222004696726799, -0.2613486051559448, 0.5350033640861511, 0.04246513918042183, -0.5834996104240417, 0.3578767478466034, -0.9448713064193726, -0.9362075328826904, 2.083305597305298, 1.168973445892334, -1.1893702745437622, 0.8531519174575806, -0.22227831184864044, -0.6611782908439636, 1.557335615158081, -1.1757346391677856, -1.5227755308151245, -0.6039319634437561, 1.6499003171920776, -2.370959997177124, -0.2868214547634125, 0.7021716237068176, 0.7164183259010315, -0.8168716430664062, 0.35939502716064453, 0.47850486636161804, 0.09313654899597168, 0.14880870282649994, -1.3321733474731445, 0.2451324760913849, 1.1490886211395264, 0.16858035326004028, -0.04937963932752609, -0.5562028288841248, -0.921130359172821, -0.49899473786354065, 0.6432260870933533, 0.7399080991744995, 1.5344384908676147, -0.09568493068218231, 0.7624797224998474, -0.2297121286392212, -0.3894725739955902, -1.6271101236343384, -0.2703935503959656, 0.023147927597165108, -1.975286841392517, 1.4794082641601562, 0.05860164761543274, -1.1143244504928589, 1.329796314239502, 0.06266528367996216, 1.3144217729568481, 0.8833491206169128, -0.20920473337173462, -0.41359275579452515, 1.072675347328186, 2.6069226264953613, -1.0139013528823853, -0.33493924140930176, -0.1472742110490799, -0.34583038091659546, 1.3963232040405273, -2.000715732574463, 0.1637258231639862, -0.809829592704773, 0.7525348663330078, 0.10557477176189423, 0.7747476100921631, 0.5297678709030151, 1.6789084672927856, 0.7363102436065674, 0.9056556224822998, 0.6275277733802795, -1.1255184412002563, -1.1042040586471558, -0.3718646764755249, -0.9044852256774902, -1.4091970920562744, -0.48704856634140015, 1.0578413009643555, 1.3486013412475586, 0.9902215600013733, 0.6943559646606445, 1.6642463207244873, -2.203881025314331, 1.7155767679214478, 1.3213833570480347, -1.0399991273880005, -0.6862095594406128, -1.0939253568649292, 1.7572849988937378, 1.3900169134140015, -0.33386579155921936, 0.3106643259525299, -0.5356333255767822, -0.010475284419953823, 0.2650395333766937, -0.36901015043258667, 0.6537225842475891, 0.2568357586860657, 0.3068596422672272, 0.08357683569192886, -0.6349881887435913, -0.4669756293296814], [0.8850992321968079, 0.5286805033683777, 0.8689478635787964, 0.6420431137084961, 1.7648595571517944, -0.735386848449707, 1.6445454359054565, -0.579990804195404, 0.1974976509809494, 0.16334566473960876, -1.8110263347625732, 0.13893382251262665, 0.24285238981246948, -0.366366446018219, -1.3544626235961914, -0.2713579833507538, -1.218735694885254, -0.5570309162139893, 0.9608163237571716, -2.214750289916992, 0.13371652364730835, -1.6347789764404297, -0.5725126266479492, 0.6523431539535522, 1.591626524925232, 0.6647078394889832, 0.4055003523826599, 1.2829071283340454, -0.05903242155909538, 0.07644271850585938, 0.14983469247817993, 0.776716411113739, -0.1694176346063614, 0.3864346146583557, -0.8065412044525146, -1.6711373329162598, 1.7495296001434326, -1.3146501779556274, -3.0009641647338867, -1.1378190517425537, 0.28087979555130005, 0.9018028974533081, 0.6937527060508728, 1.3810782432556152, 0.957517147064209, -0.6387594938278198, 0.2694505751132965, 1.130252718925476, -1.3691225051879883, -1.3073103427886963, -0.836344838142395, 0.16257154941558838, -0.7420853972434998, 0.5948907732963562, -0.562741756439209, -0.6185738444328308, -0.314914345741272, -1.345479130744934, 0.32048237323760986, 0.9454467296600342, 1.3995018005371094, -0.12250054627656937, -1.024654746055603, -0.23642733693122864, 0.2143067568540573, -2.202511787414551, 0.09987431019544601, 1.5684130191802979, 0.8065282702445984, 0.1363227665424347, -0.4383928179740906, 0.9147171378135681, -1.2041566371917725, 1.0572199821472168, -1.362446904182434, -0.7296232581138611, 1.4908345937728882, 0.12084552645683289, -0.11519819498062134, -0.6495336890220642, -1.0253666639328003, -0.2945949137210846, 0.4364413917064667, 0.7392673492431641, -0.3113986551761627, 0.14095915853977203, 0.09290292859077454, 1.5781073570251465, 0.5728262066841125, 0.8123517632484436, 0.8339624404907227, 0.2691124677658081, 2.2863922119140625, -0.9091733694076538, 0.634943425655365, -0.2948901057243347, -1.3943302631378174, 0.0529462993144989, 0.13447777926921844, -0.3996135890483856, 1.8692270517349243, -1.468569278717041, 0.6597849726676941, -0.3745601773262024, -2.0715439319610596, -2.0029969215393066, -0.7555241584777832, -1.0891473293304443, -1.3767297267913818, 1.1750922203063965, 1.3370405435562134, -0.8820752501487732, -0.18145258724689484, 0.8347383737564087, 0.6347784399986267, 1.111886978149414, 0.053487103432416916, -0.756413996219635, -2.326230049133301, -0.7936902046203613, 1.0184942483901978, 0.03123595006763935, 0.01445510983467102, -0.2619287073612213, -0.28793373703956604, 1.2162154912948608, -0.3412877023220062, 0.38045617938041687], [-0.40247076749801636, 0.7050389051437378, 0.4647284746170044, -0.049620650708675385, 1.2769240140914917, 0.4174407422542572, 0.5267298221588135, -2.204819440841675, -0.5735340714454651, 0.6535641551017761, -0.5120236277580261, 0.9148529767990112, 1.1265714168548584, 1.144731044769287, -1.1254204511642456, 1.2779998779296875, -0.8648334741592407, 0.7678146362304688, 0.5801665186882019, 0.03062237985432148, -0.2739723324775696, 0.4389873445034027, -1.84566330909729, -0.43597522377967834, 0.5560891032218933, -0.4010792374610901, 0.36865130066871643, -0.000985527760349214, -0.06966947764158249, -0.05920213460922241, 0.154587522149086, 0.25181078910827637, 0.2309584766626358, -0.9223660230636597, -0.5635623931884766, -1.512200951576233, 0.6306386590003967, -0.24386988580226898, 0.007616135757416487, 2.2169225215911865, 1.041234016418457, 0.0041619581170380116, -0.20398280024528503, -1.6066275835037231, -0.027625560760498047, 0.6254021525382996, -0.7580428123474121, 0.9075940251350403, -0.2698226571083069, 0.6127871870994568, -0.04539920389652252, -1.7614898681640625, -1.5379287004470825, 0.3349890112876892, 1.0188586711883545, 0.028669677674770355, 0.8001497387886047, -0.08129678666591644, -0.5846346616744995, -0.2921786904335022, -0.8262592554092407, -1.010221242904663, -0.3576492369174957, -0.4041513502597809, 0.25075846910476685, -0.9899715185165405, 0.22246579825878143, 0.21760232746601105, 0.4136122465133667, 0.40055033564567566, 0.2468847632408142, -1.3847085237503052, -1.2263306379318237, 0.8893405199050903, -1.3161096572875977, 1.5707083940505981, 0.407284677028656, -0.8047165870666504, -0.19513505697250366, -0.08807256072759628, -0.6483258605003357, 1.1451613903045654, -0.33087921142578125, -0.27948594093322754, 0.5241959691047668, 2.2234792709350586, -0.7710702419281006, -0.9341614842414856, 1.0588068962097168, -1.2375930547714233, 1.2121022939682007, -0.413492888212204, 0.3853154480457306, -1.5793591737747192, -0.4587918519973755, 1.656246304512024, 0.09201611578464508, -1.1780543327331543, 0.6444993019104004, -0.6662448048591614, 0.7964503765106201, 1.169634461402893, 0.8636000752449036, -0.3072330355644226, 0.7376287579536438, -1.7842183113098145, 0.41921186447143555, 0.9732400178909302, -0.6862853765487671, -1.2421895265579224, 0.6263681650161743, -2.0457684993743896, 0.8709143996238708, 0.14546045660972595, 0.7827115654945374, -0.7669579982757568, 0.15098097920417786, -1.154517650604248, 0.33466285467147827, 0.014339687302708626, 0.3082050681114197, -1.2653433084487915, -0.17103081941604614, -0.6261332035064697, -0.41057440638542175, -1.3452434539794922, 0.4999348223209381, -1.220991611480713], [0.17983640730381012, 0.8114235997200012, -1.4515953063964844, -2.1070878505706787, -1.2618211507797241, -0.30121615529060364, 1.0022579431533813, -0.821405291557312, -1.8701058626174927, -0.8224936127662659, -2.4057204723358154, 0.012272097170352936, 0.3681175708770752, 0.7624445557594299, -1.0361039638519287, -1.4063141345977783, 0.1920076161623001, -0.23096996545791626, 0.8415234088897705, -0.6539611220359802, 0.48613542318344116, -1.114633560180664, -0.7139500379562378, -1.7391771078109741, -0.15570758283138275, 1.1677656173706055, 0.7230793833732605, -0.22793720662593842, 0.3224608600139618, -0.7986159324645996, -0.9822906851768494, 1.762001872062683, 0.1745402067899704, -0.018292861059308052, 0.043459076434373856, -0.38777589797973633, 0.889518141746521, -0.05537937954068184, -0.6374800205230713, -0.02199222333729267, -0.8874300122261047, -1.526793360710144, 0.7749478816986084, 0.03850467503070831, 0.6600638031959534, 1.7167093753814697, 0.7098115682601929, 1.5603337287902832, -1.9401814937591553, -0.7014212012290955, -0.9961686134338379, -0.8830171823501587, 1.8845720291137695, 0.5243831872940063, 0.5638307929039001, 0.8511644601821899, -0.32742810249328613, -1.054770588874817, -0.16179881989955902, -0.9942417740821838, -0.18880894780158997, 1.187142014503479, -1.302049994468689, -0.5954400897026062, -0.9220054149627686, -0.6142706871032715, 0.4932188391685486, 1.0447465181350708, -1.0160529613494873, 0.10138825327157974, 1.8177322149276733, -1.496119737625122, 0.1621619313955307, -0.47443515062332153, -0.15886570513248444, -0.8960151672363281, 1.6058201789855957, -1.0250945091247559, -0.6527813076972961, -1.2043043375015259, 0.11198198795318604, 1.1264336109161377, 1.3049622774124146, -0.8426914215087891, -1.196976900100708, 0.5191958546638489, 1.7184816598892212, 1.0095077753067017, 1.5038951635360718, -0.703671395778656, 0.0931754782795906, -0.22415056824684143, -1.9595880508422852, 0.3059004247188568, 0.9074344635009766, -0.5886135101318359, -0.2764226496219635, 0.6603103280067444, 0.30912351608276367, 2.344944477081299, -0.0005100545822642744, -0.6466380953788757, 0.36346468329429626, 0.14833472669124603, -0.3706783652305603, -0.7882305383682251, -0.19979873299598694, 2.1047918796539307, 1.9190804958343506, 1.4236849546432495, 1.338339924812317, -0.15168891847133636, 0.8359988331794739, -0.5811471939086914, -1.363901972770691, 1.7471745014190674, -2.2048985958099365, 1.1111685037612915, 0.8812979459762573, -0.02938631922006607, -0.7913244366645813, 0.11986186355352402, -0.667799174785614, 1.7724382877349854, 0.1768869310617447, 0.1263953298330307, -0.5378066897392273, -0.5501641035079956], [0.45145881175994873, -0.8024853467941284, -1.6812547445297241, 1.4585628509521484, -0.10748164355754852, 1.35025954246521, 1.4749056100845337, -0.6373506188392639, -0.8065494894981384, -1.7959064245224, -1.730602502822876, 0.3002772927284241, -0.7585769891738892, -2.1173667907714844, 0.7637004852294922, -0.34446120262145996, 0.46638724207878113, -0.41647541522979736, 2.587184429168701, -0.050271324813365936, -1.2465260028839111, 1.201189398765564, 0.7284986972808838, -1.9742497205734253, -0.38716861605644226, -1.6272475719451904, -0.6020013689994812, -0.3719242811203003, 0.9112023711204529, -1.2853997945785522, 0.20485010743141174, 0.8033133149147034, 1.4952749013900757, -0.9044205546379089, 0.5271101593971252, -0.7754635810852051, 1.3687843084335327, -0.0036056083627045155, -1.1203325986862183, 0.3127640187740326, 0.6927917003631592, 0.4447643756866455, -1.9060312509536743, 0.2291562557220459, 1.8701744079589844, -0.2195487767457962, -0.08215557783842087, 1.8925105333328247, -0.5629058480262756, -1.190555453300476, 0.5745882987976074, -0.04991242662072182, 0.39264827966690063, -0.2513982951641083, -0.5095597505569458, -0.10712248086929321, 1.468181848526001, 1.146549105644226, -0.12625627219676971, 0.9579464793205261, 0.5413140654563904, -1.39197838306427, -0.8036931753158569, -0.1971176564693451, 0.7561429142951965, 1.3407304286956787, 0.5997816324234009, 1.1086108684539795, 0.7366309762001038, 1.4530903100967407, -1.6444681882858276, -0.3796483874320984, 0.7469433546066284, -0.13452161848545074, -1.5073878765106201, 0.38131964206695557, -1.0355738401412964, -0.08510482311248779, -0.7739015817642212, -0.03678840771317482, -1.4780045747756958, -1.0423486232757568, 1.4537736177444458, 0.7113146781921387, 1.0213874578475952, 0.3301173448562622, 0.4186212122440338, -0.07209673523902893, 1.5687596797943115, 0.7917729616165161, -0.011955322697758675, 0.14067262411117554, -0.3797004818916321, -1.0572023391723633, -0.02816089428961277, 0.27718403935432434, 0.3290617763996124, -1.179874300956726, -0.47261905670166016, -0.951472818851471, 1.5061192512512207, 0.7455956339836121, -1.809942364692688, 0.8851211071014404, -1.0223592519760132, -0.14406505227088928, 0.048103757202625275, -0.044191569089889526, -0.6082586646080017, 0.18686921894550323, -0.5717470645904541, -0.4111649692058563, 0.8339163661003113, 2.3267478942871094, -0.22625626623630524, 0.885214626789093, -0.9756794571876526, 1.0637215375900269, -0.4679264724254608, 1.051301121711731, -0.6749570965766907, -0.8170035481452942, -0.16773714125156403, -2.7379090785980225, 0.7424992322921753, 1.0348167419433594, 0.5960706472396851, 1.4048875570297241], [0.7594674229621887, 1.786660075187683, 1.138571858406067, 0.01088176853954792, 0.28767165541648865, -0.28999581933021545, 0.22159095108509064, -1.8956804275512695, 0.5513370037078857, 2.06270694732666, -0.456340491771698, 0.6766911149024963, -0.11474812775850296, 0.15110397338867188, -1.7537319660186768, 0.02088615857064724, 0.30812227725982666, -2.020993232727051, -0.6529929637908936, 0.7823874950408936, 0.06544281542301178, -0.06281684339046478, -0.4865894317626953, -1.9388997554779053, -0.8672545552253723, -1.1062870025634766, 0.21590711176395416, 0.6110736131668091, -0.6514177322387695, 0.4443950653076172, -0.06251504272222519, 0.9349879622459412, -0.5681464076042175, -1.3417574167251587, -1.1788831949234009, 0.8973009586334229, 0.6837441325187683, -0.8530069589614868, -1.5126897096633911, 0.2358139157295227, 1.0865315198898315, -0.11802871525287628, -1.049816370010376, -0.20066829025745392, 1.0529519319534302, -2.366326332092285, 0.28431108593940735, 0.0779903456568718, -1.225426197052002, -0.10350138694047928, 1.2117741107940674, 2.57564640045166, 0.5209547877311707, 1.0155824422836304, -1.2124320268630981, 0.5010269284248352, 0.10217426717281342, 0.724388062953949, -1.7343593835830688, -2.002938747406006, 0.17932990193367004, 1.9069582223892212, -0.5787001848220825, 1.6415811777114868, -0.3282712697982788, -1.235344409942627, 0.33217546343803406, 1.191973090171814, 0.071429044008255, 0.06892271339893341, 0.03828338906168938, -2.2654497623443604, -0.4758862257003784, -0.2750108540058136, -0.4422568380832672, -1.7058897018432617, -0.815676212310791, 0.2679287791252136, 0.9210711717605591, 1.3208036422729492, -0.837392270565033, -0.14688041806221008, -1.7762070894241333, 0.6810738444328308, 0.5677944421768188, 1.611080288887024, -1.2383232116699219, -1.1011534929275513, -1.6295629739761353, -0.4124632477760315, -0.36909446120262146, 1.012393832206726, -0.4986778795719147, 2.035832166671753, -0.5139064192771912, -1.0818250179290771, 0.6748130917549133, -0.34446170926094055, -0.6447256803512573, -0.8316686749458313, 0.20613059401512146, -0.08014032989740372, 0.8916856050491333, -0.014120583422482014, 1.6291687488555908, 0.5761790871620178, -0.6200485229492188, -1.4335007667541504, -0.44596758484840393, -0.7597429752349854, -1.5525175333023071, -0.31713563203811646, -2.5085082054138184, -0.3961022198200226, 0.1502181887626648, 1.1947599649429321, -0.23220236599445343, 0.8222458958625793, 0.5988042950630188, 0.08933757245540619, 0.5824880599975586, -0.49389883875846863, -0.4065026342868805, 1.1147658824920654, 0.48080363869667053, -0.6029148101806641, 1.0328431129455566, -0.2018447071313858], [-0.4288385510444641, 0.45843952894210815, 1.5147089958190918, 0.09411876648664474, -1.1412711143493652, 0.6168781518936157, -1.7053149938583374, 0.33075806498527527, -0.6011328101158142, 0.5707609057426453, 0.13541820645332336, 0.9665666222572327, 1.1805237531661987, -0.21251657605171204, -0.4097997546195984, 2.0111653804779053, 1.6167649030685425, 1.09841787815094, -1.701141119003296, -2.3221867084503174, 0.25121209025382996, 0.9902663826942444, 0.8965124487876892, -1.300794243812561, -0.5591216087341309, 1.060726523399353, 0.19480657577514648, 0.01631161943078041, 0.8480908870697021, 0.13570889830589294, 0.09770683199167252, -1.6663519144058228, 0.17231886088848114, -0.631532609462738, -0.6122399568557739, -0.05976039916276932, 0.9212742447853088, 1.3985179662704468, 0.37013939023017883, -0.027326062321662903, -0.1449383944272995, 0.4699151813983917, -0.7404915690422058, 0.8411062955856323, 2.5615782737731934, 0.381664514541626, -2.274453639984131, -1.5099289417266846, 1.4066431522369385, -0.28298649191856384, -1.326179027557373, -1.6361724138259888, -1.6058447360992432, 1.4780702590942383, -0.06882055848836899, 0.9883670806884766, 1.4376585483551025, -0.1912011355161667, 2.2557921409606934, 0.9046629071235657, -0.08640112727880478, -3.269139289855957, 0.13835011422634125, -1.6958112716674805, -0.6762158870697021, 0.5310284495353699, 0.734212338924408, 1.406038522720337, 1.2815501689910889, -1.61508309841156, -0.21160465478897095, 2.1203725337982178, 0.2528650760650635, -0.9238028526306152, -1.0045865774154663, -0.5363798141479492, -0.7805497646331787, 1.0116965770721436, -0.31991076469421387, -0.2355232834815979, -1.6584813594818115, -0.3794320225715637, -0.9173640012741089, 0.05334426462650299, 0.8957672715187073, 0.8818550705909729, 0.5294235348701477, -0.5356714129447937, 1.0807099342346191, 1.7496886253356934, -0.5362966060638428, 1.2905925512313843, 0.82881098985672, -1.3738195896148682, 0.8555617332458496, 1.036695957183838, 0.05006558820605278, 0.17389973998069763, 2.120009422302246, -1.3925727605819702, 0.8784825205802917, -0.9639129042625427, 0.11435549706220627, -2.0351853370666504, -0.41536304354667664, -1.7792521715164185, 1.2518224716186523, 0.3827633857727051, 0.47350260615348816, 0.4447956085205078, 1.3297419548034668, 0.3494575619697571, 0.6102651357650757, -1.0905797481536865, 0.6403843760490417, 1.488165259361267, -1.8714735507965088, 1.2954457998275757, 0.7113491296768188, 2.151590585708618, 0.12226110696792603, -1.8738459348678589, 0.38215774297714233, 0.7661396861076355, -0.957943856716156, -0.12947142124176025, -0.8496679663658142, -0.27559998631477356], [-0.5320276021957397, -0.4039452075958252, -0.06054956838488579, -0.3662797510623932, 1.7597440481185913, 0.49131718277931213, -0.6166833639144897, -2.918527603149414, 0.47681373357772827, -0.8635560274124146, 0.8852725625038147, -0.8140056729316711, 0.3983913064002991, 1.8599250316619873, -1.3584811687469482, -1.2505314350128174, 0.5714081525802612, 2.8022704124450684, 1.9036530256271362, -2.1746463775634766, 1.4508812427520752, -0.7335827946662903, 0.3000681400299072, 1.231185793876648, 0.5455114245414734, -0.6231940388679504, -0.8872793912887573, -0.2717525362968445, -0.5672298669815063, -0.5130577683448792, 0.7535818815231323, 0.9354464411735535, -0.3138478994369507, 0.6964871883392334, -1.3166966438293457, -1.3654217720031738, -0.8776363730430603, -0.2218049317598343, -1.168105959892273, -0.21168194711208344, 0.9215131998062134, 1.2909841537475586, 0.5533900856971741, -1.3006854057312012, 0.25543588399887085, -0.2964540719985962, -0.20801451802253723, 1.3411099910736084, -1.5189926624298096, -0.6277002096176147, 0.20655900239944458, 2.38588285446167, 0.7312127351760864, 0.06654275953769684, 0.8378162384033203, -0.4700811505317688, -0.9202622175216675, -1.3195226192474365, -0.12911179661750793, 0.48397818207740784, -1.094059944152832, 0.5973963737487793, 0.19152843952178955, -0.669238805770874, 0.2662239372730255, -0.6062288880348206, 0.7678468823432922, -1.098924994468689, 0.8170390129089355, -1.7462960481643677, -0.6226941347122192, 0.6962659955024719, -0.45905938744544983, -1.0121132135391235, 0.694911777973175, -0.18924923241138458, 0.6078631281852722, -0.645259439945221, 1.3598978519439697, -1.0682239532470703, -0.5829082131385803, 0.26736411452293396, -1.2090855836868286, 0.10423848032951355, 0.11041120439767838, 0.2240578681230545, 1.3329792022705078, -0.8841721415519714, 0.8141810297966003, 0.11464507132768631, -1.40066397190094, 0.7392211556434631, 1.553688883781433, 1.3246582746505737, 1.1329412460327148, 1.0946582555770874, 0.08789772540330887, 0.15456169843673706, 0.3372528851032257, 0.4191930294036865, 0.31826457381248474, -0.46316125988960266, -0.7159301042556763, -1.2101374864578247, 0.2643514573574066, 0.04037405550479889, 0.13053685426712036, -1.1712521314620972, 0.048580557107925415, 0.5260931849479675, -3.3188915252685547, -0.6736699938774109, -0.1684463620185852, 0.36893120408058167, -0.18165870010852814, 0.30145689845085144, -1.2253310680389404, 0.5391043424606323, 0.27389055490493774, -0.6573556065559387, -0.48489895462989807, 0.004275510087609291, 0.27043697237968445, 1.007715106010437, 0.5526835918426514, -1.212685465812683, -1.109268307685852, -0.21201568841934204], [1.1388769149780273, 0.3066249489784241, 0.2993684411048889, 0.10624997317790985, -1.0134090185165405, -1.0467172861099243, 2.416402578353882, 1.470818042755127, -0.44078195095062256, -1.442833662033081, -1.4790321588516235, -0.5954346060752869, -0.4655289053916931, 0.8334267735481262, -0.6620232462882996, -0.20410507917404175, -1.2586126327514648, 3.506387233734131, 0.046615373343229294, 1.221761703491211, -0.17152881622314453, 1.163325548171997, -0.5714741945266724, -2.584460735321045, 0.36169761419296265, -0.8166435360908508, -0.9954070448875427, 0.3497762382030487, -1.5969884395599365, 0.06136832386255264, -0.3927527666091919, -1.5584678649902344, 0.7042555809020996, 0.23851804435253143, 1.4036628007888794, -0.02817625179886818, 0.11402624845504761, -1.89210844039917, 1.37606942653656, -0.22022350132465363, 0.3023419976234436, -0.6486893892288208, 0.4816665053367615, -0.2966707944869995, 0.35033610463142395, -0.42465487122535706, -1.143200397491455, 1.4351732730865479, -0.029986711218953133, 0.07147163152694702, -0.34661704301834106, -0.873610258102417, -1.0331193208694458, 1.1282049417495728, -0.26865577697753906, -0.7375953793525696, 0.7437908053398132, 0.8069785237312317, 0.6325610280036926, -0.7955195307731628, -0.6191261410713196, 0.4371333420276642, 0.3826337158679962, 0.91144198179245, 1.9268232583999634, -0.31386440992355347, 1.7351961135864258, -0.0739247277379036, 1.5652360916137695, 0.10011209547519684, 0.15336526930332184, 0.1952826827764511, 1.3614983558654785, -0.6449835896492004, 1.0142349004745483, -0.07607141882181168, -0.8742741942405701, 0.06755862385034561, 1.041377305984497, -0.23429091274738312, 0.4221493601799011, -0.14561833441257477, -0.6930682063102722, -0.6471680998802185, 1.4133045673370361, 1.5779446363449097, 1.142539381980896, 1.8553465604782104, -0.5566708445549011, -0.7863161563873291, -0.7233871221542358, -0.1911851018667221, 1.1216504573822021, 0.9940618276596069, -2.4914517402648926, -1.0547841787338257, -0.5410569310188293, -0.3845423460006714, -1.0406978130340576, -1.5494954586029053, 1.2590445280075073, 0.6404863595962524, -1.5350135564804077, 0.31798890233039856, 1.168419361114502, -0.8221219182014465, 0.12464167922735214, -0.7128528356552124, 0.0892946869134903, -0.1644621640443802, -0.24298183619976044, 0.4180961847305298, -0.30708369612693787, -0.07467953115701675, 0.7580147981643677, 0.6284489631652832, -1.2611565589904785, -1.1896363496780396, 3.240811824798584, 1.4183253049850464, 2.24988055229187, -0.818076491355896, 0.43611985445022583, 1.3417823314666748, -0.3734663724899292, 1.68382728099823, -1.4660953283309937, -0.8826000690460205], [-0.2514302432537079, 0.4076828062534332, 1.413799524307251, 0.2953275144100189, -0.4225691854953766, 0.3281460404396057, 0.6581742167472839, 0.7901555299758911, -0.8885655999183655, 0.30518898367881775, 0.6458919644355774, 0.08226275444030762, 0.14034925401210785, 1.3164021968841553, 1.7151395082473755, 0.9661809802055359, 0.7302525043487549, 1.345439076423645, -1.2651019096374512, 1.3751009702682495, -0.6649725437164307, 0.25176501274108887, 1.6668449640274048, -0.36498549580574036, -0.41403618454933167, -0.6422683596611023, -0.37870511412620544, 0.8864216208457947, -0.6261876225471497, -0.27835431694984436, 0.22057242691516876, -0.9487342238426208, 0.24422836303710938, 0.17050346732139587, -0.8079922199249268, -0.17600084841251373, -0.7640542387962341, -1.020065426826477, -0.41521209478378296, 1.093461275100708, -1.101949691772461, 0.7208095192909241, 0.9630027413368225, -0.4002901613712311, 0.573775053024292, 0.30970898270606995, -1.1120920181274414, 1.178383231163025, 1.4426573514938354, 1.9436148405075073, -0.5700956583023071, -0.44734033942222595, -0.24145832657814026, 0.3546142876148224, -0.3180212676525116, -0.8822001218795776, 0.37193313241004944, -0.8090246915817261, 2.8261656761169434, 1.8451120853424072, 0.06460494548082352, 1.229999303817749, -1.2578139305114746, -0.06509213894605637, 0.8906994462013245, 1.315298080444336, 0.7159479856491089, -0.766464352607727, -0.6809656620025635, 1.0438212156295776, -0.8807698488235474, -0.029305867850780487, -1.1451573371887207, 0.23712636530399323, 0.6836196184158325, 0.2883722186088562, 0.6227049827575684, 2.029327154159546, 0.7769667506217957, 0.28048476576805115, -0.11263148486614227, -1.4961267709732056, -1.0358679294586182, 0.17604559659957886, 0.664955735206604, -0.9488399624824524, -2.221204996109009, 1.6198539733886719, -0.5714133381843567, 1.5953936576843262, 1.453827142715454, 0.18182192742824554, 1.2040125131607056, -0.2599121034145355, 0.29736852645874023, 0.5689339637756348, 1.6636606454849243, -0.3110354244709015, 0.018359677866101265, -1.1766818761825562, -0.8069188594818115, 0.0590425543487072, 0.8205428123474121, 0.6728489995002747, -0.4818021059036255, 0.24869006872177124, 1.9574286937713623, 0.4553816616535187, 2.0392889976501465, 0.8271336555480957, 1.629084587097168, 0.6166686415672302, -0.5559636354446411, -0.13143402338027954, -0.515610933303833, 0.6797988414764404, 0.16118192672729492, -0.6239427328109741, 1.3278990983963013, -0.2454642653465271, -0.12648490071296692, 1.1751703023910522, -1.1808561086654663, -0.052101898938417435, -0.7599292397499084, 0.5248681306838989, 0.3528382182121277, -0.19657377898693085], [0.1454109400510788, 0.03234110400080681, -0.05656491592526436, 0.5456873178482056, -0.02677222713828087, 0.6106936931610107, 0.2574405074119568, -1.0573915243148804, -1.255033016204834, -0.7214955687522888, 0.9661771655082703, 0.7786125540733337, 1.6364009380340576, -0.7215110063552856, 0.2298966646194458, -0.7443014979362488, -0.5447147488594055, 0.16017377376556396, 1.1287301778793335, 0.6049492359161377, 0.06274951249361038, 0.9068374037742615, -0.0725053995847702, 2.000063180923462, -1.1298755407333374, 0.5161155462265015, -0.7607554197311401, -1.711510181427002, -1.1719547510147095, -0.05263933539390564, 0.4293544888496399, -1.8512073755264282, -0.5281819701194763, 1.3468458652496338, 0.8326259255409241, -0.3323451280593872, -0.18454806506633759, -0.42840880155563354, 1.2838164567947388, 0.8308687210083008, -1.1755868196487427, -0.30649811029434204, 0.12389247864484787, 1.986555814743042, -0.4062986969947815, 1.4021888971328735, -0.4283938407897949, -0.31930357217788696, -1.4376851320266724, 0.37653276324272156, 0.17661738395690918, -0.2673412263393402, -0.2540741562843323, 2.644080877304077, -0.014702304266393185, 1.6497479677200317, -0.5110073089599609, -1.3837471008300781, -0.09857312589883804, 0.7529525756835938, 1.3944871425628662, 0.5796160697937012, 0.4031241536140442, -0.407610684633255, 1.7114838361740112, 0.0772111788392067, -0.20928052067756653, 1.9937632083892822, 1.2207928895950317, -1.0511257648468018, -0.8460864424705505, -0.9830548167228699, -0.01879044808447361, -0.6276236176490784, -0.7656641006469727, 1.1275944709777832, 0.504551351070404, -0.014278477989137173, 1.4066331386566162, -0.14858850836753845, -0.3759588301181793, -0.33013424277305603, 0.8113874197006226, 1.1817094087600708, 0.6614319682121277, -1.334897756576538, 2.0108108520507812, -0.4126816689968109, -0.3653324246406555, 1.1300809383392334, -1.853506088256836, 0.6460473537445068, 0.1517719328403473, 0.7646132707595825, 0.5177063941955566, -1.5664255619049072, 1.1128218173980713, 2.466095209121704, 0.5115083456039429, -0.6681011319160461, 0.03153350576758385, -0.6202012896537781, -1.326961874961853, -0.17309284210205078, -0.902642011642456, 1.098430871963501, 1.056947946548462, -0.19253630936145782, -1.1646028757095337, -0.1710747927427292, 0.03793532773852348, 1.1622540950775146, 0.3759293556213379, -0.15587389469146729, -0.7683455944061279, -1.3976927995681763, -0.8206504583358765, 0.9651226997375488, -1.3603582382202148, 0.34894534945487976, -1.2111470699310303, -0.7071636915206909, -1.3923773765563965, -0.3078814446926117, 1.236769676208496, 0.09757710993289948, 0.17566880583763123, -0.7539161443710327], [0.4482976794242859, 1.5009031295776367, 0.5424546599388123, -0.985034704208374, -1.6390070915222168, -0.057205867022275925, -0.07372595369815826, 2.6311745643615723, 0.12911765277385712, -0.33051979541778564, 0.6976329684257507, 1.0132580995559692, -1.8477689027786255, -0.5090176463127136, 0.39788269996643066, -0.8178654313087463, -0.5294228792190552, 1.88243567943573, 1.521846055984497, 0.9150562286376953, -1.2872228622436523, -0.4088321030139923, -0.5866715908050537, -1.5507915019989014, -0.15127632021903992, -1.0739762783050537, -0.8870276212692261, 1.470355749130249, 0.2974534332752228, -1.5477510690689087, 1.470726728439331, -1.453963041305542, -0.47212207317352295, -0.5787002444267273, 1.7407950162887573, 0.33604371547698975, -0.485551655292511, 0.6299763917922974, -0.9719445705413818, 0.20592458546161652, -0.5182716846466064, -1.8653948307037354, -0.39280977845191956, -2.4498794078826904, 0.0065536703914403915, -0.47091901302337646, -0.36205577850341797, -0.45624488592147827, 0.2132415622472763, 0.6150904297828674, -0.520427942276001, -0.9821168184280396, -0.7369136810302734, -0.504872739315033, -0.8497076630592346, -1.1102336645126343, 0.5071865320205688, -0.032657917588949203, 1.2970176935195923, -1.3379712104797363, -0.7796623110771179, 0.9812655448913574, 0.05467107519507408, -0.982276201248169, -1.5922648906707764, 0.6670301556587219, 1.2546535730361938, -1.3623130321502686, 1.4523862600326538, 1.0746663808822632, 0.8233188986778259, 1.5345983505249023, 0.38905277848243713, 0.028871744871139526, 1.6456902027130127, 0.5574495792388916, -0.4870201349258423, -1.6414834260940552, 0.4331054389476776, 0.6659305691719055, 0.13635201752185822, -0.04843491315841675, 0.5553447008132935, 0.0921858698129654, -1.2695069313049316, 0.056321483105421066, -0.8697268962860107, -1.8016499280929565, -0.20300708711147308, 0.7515652775764465, 0.44360268115997314, -0.7785834074020386, -0.8375321626663208, -0.7939634323120117, 0.9593107104301453, -0.5379470586776733, -1.2582744359970093, 1.7260924577713013, 0.5710985660552979, 0.11414320766925812, 1.8830355405807495, 1.4821220636367798, 0.6184993982315063, 0.787998378276825, 0.38950878381729126, -0.47927504777908325, 0.5423405766487122, -1.0958786010742188, -0.01953326165676117, -0.8137505054473877, 0.44231557846069336, 0.6384865045547485, -1.0810757875442505, 0.893101692199707, -0.11420014500617981, -2.300583839416504, 0.3696352243423462, 0.30713099241256714, 0.7303984761238098, 2.0374233722686768, -2.083444595336914, -0.48679473996162415, -0.10698602348566055, -0.1000324934720993, 0.5086899995803833, -1.0034821033477783, -1.0973244905471802, -1.3685688972473145], [-1.598559856414795, 1.686253547668457, 0.8744266629219055, 0.3262680470943451, 1.4021968841552734, -0.9558638334274292, 0.4237912595272064, 1.5781474113464355, -1.5488767623901367, 1.5474191904067993, 0.20556381344795227, -2.570470094680786, -0.9917207956314087, 1.5271632671356201, 1.1519898176193237, -1.7388533353805542, -0.0868179127573967, -0.037785615772008896, 0.5618952512741089, -0.5833553671836853, -0.2707309126853943, 0.3532103896141052, -1.4310312271118164, -1.1471238136291504, 2.2453091144561768, -0.011338516138494015, -1.5587053298950195, 1.3739134073257446, 0.8483996987342834, -1.6939421892166138, -0.3438645899295807, 0.6793661117553711, 0.34622105956077576, 0.5124883055686951, 0.578399658203125, -0.5990514755249023, 0.8791600465774536, 1.0563572645187378, -0.3875914216041565, -0.10691382735967636, -0.9693209528923035, 0.4171864688396454, -0.6680940985679626, -1.165016531944275, 1.014747977256775, 0.6202700138092041, -1.0009682178497314, 1.0353596210479736, 0.6914085149765015, 0.3772064745426178, -1.4653536081314087, -1.0627552270889282, 1.2730629444122314, 1.867655634880066, 0.9540593028068542, -0.6665012836456299, -1.7674692869186401, -1.444352626800537, 1.3923877477645874, 1.4790815114974976, 0.7695518136024475, 0.6267125010490417, 0.2720564007759094, -0.17121398448944092, 1.0678030252456665, -0.9459783434867859, -0.5811198353767395, -0.3308764100074768, 0.48289307951927185, -0.512104332447052, -0.5889146327972412, -0.4628346562385559, -0.15221109986305237, 0.09432777017354965, 0.002105538034811616, -2.0437281131744385, 0.605347752571106, -1.2539472579956055, 0.5349598526954651, 0.09082642942667007, -0.05377054959535599, -0.5367770195007324, 0.7195292115211487, 0.3293517231941223, -1.1248047351837158, 1.072157859802246, -0.6470286846160889, -1.446679949760437, -2.7171030044555664, 0.8164474368095398, 1.9249650239944458, 0.6365463137626648, 0.23308207094669342, -0.38276609778404236, -0.35942044854164124, 1.1194641590118408, -0.3260347247123718, -0.5435194373130798, -1.6587892770767212, -1.0585572719573975, -0.17864933609962463, 0.08662913739681244, 0.7408085465431213, 0.22746101021766663, -0.11144690215587616, -0.6342819929122925, 1.0780075788497925, 0.4264613687992096, -2.405297040939331, 0.39033007621765137, -1.8333863019943237, 1.1766823530197144, 0.8991080522537231, 1.36064875125885, 0.59369295835495, 0.15799777209758759, -1.41949462890625, -0.9944881200790405, 0.16204722225666046, 0.04867030307650566, -0.5390655398368835, 0.8665722608566284, 0.372676283121109, -1.1406431198120117, 0.15966059267520905, -0.5433835387229919, 0.3327323794364929, 0.34346696734428406], [0.9259183406829834, 0.7118431329727173, 0.5534571409225464, 1.434684157371521, 0.4841296374797821, -0.5393239855766296, 0.4468880593776703, 0.08818421512842178, 0.17200197279453278, 2.778571844100952, -0.6922799944877625, -1.5020517110824585, -1.2131149768829346, 0.07000412791967392, -1.901006817817688, 0.1062418669462204, -0.8269566893577576, 0.04797693341970444, -0.47043633460998535, -1.0472742319107056, -0.31877338886260986, 0.2687033712863922, 0.5904529094696045, -0.08749736100435257, -0.5847625732421875, -1.3969554901123047, 0.3896213471889496, 0.6240916848182678, 0.0911022275686264, -0.9777604341506958, -1.037322998046875, 0.04474518448114395, 0.11634168773889542, -0.9083489179611206, -1.0095051527023315, 0.7774531841278076, -0.41654494404792786, -0.30932649970054626, -0.24908389151096344, -0.7811788320541382, 0.2450648546218872, -0.066081702709198, 0.9008489847183228, -0.5171324610710144, -1.2488977909088135, 0.4452006220817566, -0.3926044702529907, -1.2032997608184814, -0.7138754725456238, -1.1551612615585327, -0.4095667004585266, -0.28870415687561035, 0.7845638990402222, 0.32358318567276, -1.9806395769119263, -0.7869843244552612, 1.2515764236450195, -0.32912498712539673, -0.3261168897151947, 0.7500221729278564, -0.7496887445449829, -0.40365391969680786, -0.5016937851905823, -0.21468490362167358, -1.0262094736099243, 0.8068956136703491, 1.380515694618225, 1.3680503368377686, -1.647094488143921, -1.2393032312393188, -0.19225119054317474, 0.9162857532501221, 0.5008055567741394, -0.8106627464294434, -0.6959147453308105, 0.8633808493614197, 1.0603870153427124, 0.5220771431922913, 0.8816876411437988, -0.5242847204208374, 1.4118810892105103, 0.2240128517150879, 0.6137387752532959, -0.25810715556144714, 0.6253007054328918, 0.5421754717826843, -0.048464324325323105, 1.2801620960235596, -1.2594764232635498, 0.0037185573019087315, -1.2786511182785034, 0.6270976662635803, -0.3818625509738922, 0.2959659993648529, 0.6683269143104553, 0.372674822807312, 1.1329704523086548, 0.3614178001880646, -0.791882336139679, -1.2440218925476074, 0.27500060200691223, -0.03582314774394035, 0.40332016348838806, -0.2172376811504364, -1.3167608976364136, -0.25893357396125793, 1.151999592781067, -0.14286063611507416, 1.004126787185669, 0.7005105018615723, -0.44712215662002563, 0.1997062861919403, -0.7065734267234802, 2.019775390625, 1.2307329177856445, 0.40727853775024414, 2.016864061355591, -0.2097845822572708, 0.721462607383728, 0.640723466873169, -1.099774718284607, 0.38374584913253784, -0.2769943177700043, 0.2687171995639801, 1.3581223487854004, 0.32460588216781616, -1.7981359958648682, -0.5410029888153076], [-0.6293008923530579, -1.4370297193527222, 0.33332785964012146, 0.10734037309885025, -0.3273523151874542, 0.5266620516777039, -1.9416823387145996, -1.017484188079834, -0.3046625852584839, 1.3243038654327393, 0.18011470139026642, 0.7673693895339966, 0.8980967998504639, -0.1356389969587326, -1.675157904624939, -0.20584948360919952, -0.07512824982404709, -0.8695507049560547, 0.7389235496520996, 0.9047579169273376, -1.9415299892425537, -1.7907222509384155, 1.600778341293335, -0.3744044005870819, -0.8365116715431213, 0.9552608728408813, 0.2194904386997223, 0.7980625629425049, -1.4349167346954346, 0.1334126740694046, -0.6622114181518555, -0.6569992899894714, -0.842120349407196, -0.6047045588493347, -1.6781196594238281, 0.09798774868249893, 0.1332244575023651, 0.1259796917438507, -1.7804003953933716, -0.0801805704832077, -0.2068757861852646, -0.06217969208955765, 0.05574282631278038, -0.7841144800186157, -0.08841858059167862, 0.19287097454071045, 0.38550496101379395, -0.6383222937583923, 0.11719398945569992, 0.46532878279685974, -0.2573550343513489, -2.0419249534606934, -0.0574374720454216, -1.409852385520935, -1.012234091758728, -1.28269362449646, -0.48672324419021606, 0.7414838075637817, -0.45379310846328735, 0.5868746042251587, -0.6975628733634949, 0.5635294318199158, -0.8614237904548645, 0.06878393143415451, -0.6248432993888855, 0.156193807721138, -0.6376727223396301, 0.7179900407791138, -0.9195365905761719, -0.09224535524845123, 1.1191375255584717, 0.9989290833473206, 1.291587233543396, -0.11694569140672684, 0.9601341485977173, -0.17763136327266693, 0.1671776920557022, 1.661508321762085, -1.0717322826385498, 0.8645772337913513, 0.8745611310005188, -0.3927841782569885, 0.74718177318573, 1.1072957515716553, -1.5520938634872437, 0.9937694668769836, 0.4265202581882477, 1.3408639430999756, -0.18583722412586212, -1.201027512550354, 0.46535539627075195, 0.33914318680763245, 0.7739090323448181, -0.20703865587711334, -1.1117422580718994, 0.17349916696548462, -0.4883281886577606, -0.8766536712646484, -0.1480531096458435, -0.5173358917236328, -0.46620944142341614, -0.26756349205970764, 1.1261861324310303, -0.5427426099777222, -0.472920298576355, 0.2959213852882385, 0.5416566729545593, 1.8837915658950806, -0.826188862323761, 0.6302873492240906, -0.9717553853988647, 1.5577958822250366, -0.6943698525428772, 1.498167634010315, 0.5438780188560486, -1.785147786140442, 0.7273029685020447, -0.3063662648200989, -0.019053438678383827, -0.5238338708877563, 2.340392589569092, 1.1387569904327393, 1.6742992401123047, 0.29252538084983826, -0.48015066981315613, -0.4755500555038452, 0.38926878571510315, -1.210754156112671], [0.18811216950416565, -1.4471333026885986, -0.42451873421669006, -0.2578589618206024, -0.15150491893291473, 2.834381341934204, -0.0309564471244812, -1.2994019985198975, 0.8531467318534851, -1.6298856735229492, 0.028090571984648705, 0.857991099357605, 1.5469577312469482, 0.9047651886940002, -1.0780811309814453, 0.9061495065689087, -0.011637510731816292, -0.14407968521118164, 1.2981334924697876, -0.3989194929599762, 0.1097443550825119, 0.17161934077739716, -0.8167262673377991, -1.8012200593948364, 1.7916231155395508, -0.0473911426961422, 0.9819075465202332, -0.5095195770263672, 0.29981690645217896, -2.305527448654175, 1.0547618865966797, 0.3501361012458801, -1.5534749031066895, -0.06654069572687149, -0.9650412201881409, 0.7652509808540344, -1.3661928176879883, -0.7364485859870911, 0.7936291098594666, 1.665554404258728, 1.7800590991973877, -0.6662124991416931, 1.4802029132843018, 0.2987017333507538, 0.7082152366638184, 0.35260993242263794, -0.448290079832077, -0.19520898163318634, 0.5626929402351379, 0.018791891634464264, -2.0653345584869385, 0.7873831391334534, -0.07517211884260178, -0.3437696099281311, 0.8857041001319885, -1.0272241830825806, -0.5718029737472534, -0.9606199264526367, 0.2813911437988281, 0.5612221956253052, -0.7461362481117249, -0.5838039517402649, 0.5800950527191162, -0.3788686692714691, -0.8297980427742004, 0.5058175921440125, 0.5034040808677673, 1.7929600477218628, 0.28758740425109863, 0.6524497270584106, -1.3546409606933594, -0.29332223534584045, 0.36257168650627136, -0.13820479810237885, 0.9307807683944702, -0.7364405989646912, 0.34559929370880127, -0.9948930144309998, 1.685731053352356, -0.15324918925762177, 1.04022216796875, -0.2953300476074219, 0.10932473093271255, -0.4638964533805847, -1.1291134357452393, -0.1709839105606079, 1.3505749702453613, -2.0640556812286377, -1.190125823020935, -1.0400400161743164, -1.1919941902160645, 0.879435658454895, -1.1462578773498535, 0.9699497818946838, -0.025301475077867508, 0.930884063243866, -0.5672661066055298, -1.4233994483947754, -0.5185144543647766, -0.7638878226280212, 0.47368496656417847, -1.1262515783309937, 0.6853868961334229, 0.3755730092525482, -0.03190240263938904, 1.0457912683486938, 0.9656731486320496, -0.4250009059906006, 2.667574167251587, -0.05161212012171745, -3.0497355461120605, 0.014133259654045105, -1.1051818132400513, -0.21660038828849792, -0.8118808269500732, -1.248081922531128, 1.5960438251495361, 1.5676488876342773, 0.80870121717453, -0.28660574555397034, 0.5146308541297913, -0.1687842607498169, 0.16063128411769867, -0.6658499240875244, 1.3368308544158936, -0.7603154182434082, -0.5157249569892883, -1.3392736911773682], [0.6311339735984802, -0.7633257508277893, 1.28204345703125, -0.06337399780750275, -0.03818228468298912, 0.8548281788825989, -0.8771178126335144, 0.009301171638071537, -1.659010887145996, -0.4686201512813568, -0.22723418474197388, -0.21292059123516083, 0.5093296766281128, 0.7847908735275269, -1.554277777671814, 0.34578484296798706, 1.0818885564804077, 0.6664729118347168, 1.5606474876403809, -0.10663309693336487, -1.0288455486297607, -0.9117984175682068, -0.2521722912788391, -0.8153417706489563, 1.8523601293563843, -0.6685522794723511, -0.5983830094337463, -0.33402618765830994, 1.8283659219741821, 0.19399355351924896, -0.8460999727249146, -0.4523298442363739, -0.3501763641834259, 0.9576064348220825, 1.8524234294891357, 1.045060157775879, -0.06592300534248352, 1.1414241790771484, -0.20384563505649567, -1.2437376976013184, 0.8784591555595398, 1.6578435897827148, 1.0748885869979858, 0.07297723740339279, 0.8839971423149109, -0.20257876813411713, 1.3421818017959595, 0.9263454079627991, 0.2949073016643524, 1.2602607011795044, -0.1873759925365448, -0.8659796118736267, 0.15043970942497253, 0.49572524428367615, -0.4072948396205902, -0.9984647631645203, 0.09989997744560242, 0.49837973713874817, 0.4292135536670685, -2.521153688430786, 0.45940130949020386, -0.5992996096611023, 0.20861199498176575, 2.1467373371124268, -1.4204559326171875, -1.2410722970962524, 0.6043877005577087, -0.1341336965560913, 1.0819036960601807, -0.050316184759140015, -0.5774322748184204, -0.17493128776550293, 1.1175239086151123, -2.3988797664642334, -0.94049072265625, 0.24844709038734436, 0.6198724508285522, 0.8213196992874146, 1.3128342628479004, 0.9455381631851196, 0.3578037619590759, -1.151244878768921, -1.1144119501113892, -1.9540841579437256, -0.6185871958732605, -1.7147071361541748, -1.22059166431427, -0.16942903399467468, -0.3859158754348755, -1.0255978107452393, -0.12124817818403244, 1.1847823858261108, -0.492804616689682, 0.12527209520339966, 0.03605371341109276, -1.0016628503799438, 0.38090401887893677, 0.06944812834262848, -0.8802851438522339, -1.0652358531951904, 0.6873892545700073, 0.027320297434926033, -0.13912546634674072, 0.012866218574345112, -0.06490755081176758, 1.2235264778137207, 0.9295039176940918, -0.13219314813613892, -0.22006161510944366, 0.6580455899238586, -1.1339436769485474, 0.9034184813499451, 0.48923367261886597, -1.6867916584014893, 0.2990156412124634, -1.1256126165390015, -0.09257760643959045, 1.138999104499817, -0.4474901258945465, -1.6268426179885864, 0.8728852272033691, -0.9989661574363708, 0.003190798219293356, 0.6311021447181702, -0.014504186809062958, -1.7752100229263306, 0.22707980871200562, 0.816156804561615], [-1.0544480085372925, 0.17644494771957397, 0.952677845954895, 0.49736499786376953, -0.7248064279556274, 2.3339741230010986, -0.6642277836799622, 1.1079680919647217, 0.16825519502162933, -0.3274517059326172, -0.8190571069717407, -0.9436972141265869, 0.28158944845199585, 0.35440006852149963, -0.01539335586130619, -0.29057127237319946, 0.6149653196334839, 0.9877046942710876, -0.5218161940574646, 0.7429559230804443, -1.6978416442871094, -3.1095855236053467, -1.122787356376648, 0.6390895247459412, 0.6353304982185364, 0.22389496862888336, 0.04928996041417122, 0.4793466329574585, -0.36219432950019836, 2.7780370712280273, -0.07804779708385468, -0.20958809554576874, -1.1390939950942993, -1.6000561714172363, -0.4553276598453522, 1.0606074333190918, 1.8492106199264526, 0.14075712859630585, 1.7502071857452393, -0.7270490527153015, 1.5358537435531616, -1.5504125356674194, 0.09156522899866104, 2.401634693145752, -0.47386831045150757, 0.8089704513549805, -0.2250405251979828, 0.6833785772323608, 0.9399593472480774, 0.5460250377655029, -0.44516703486442566, 0.3734169900417328, -1.6427085399627686, 0.014277464710175991, 0.27699244022369385, 0.662193238735199, 1.5682183504104614, 0.0062767984345555305, 0.21052737534046173, 1.2487823963165283, 1.3117969036102295, 0.1707635521888733, 1.0129508972167969, 0.6713680028915405, -0.7736542820930481, 0.436452716588974, 0.15957742929458618, -0.06228366866707802, 1.6836494207382202, -0.9008969068527222, -1.6459972858428955, 0.6283133029937744, -1.240182876586914, -1.4896985292434692, 1.049783706665039, 1.6370846033096313, -0.8075161576271057, -2.425410509109497, 1.5270971059799194, 0.4608045220375061, -0.08358390629291534, 1.7394107580184937, -1.0595563650131226, -1.5188528299331665, -0.997823178768158, -0.03279886394739151, -0.3293623626232147, -0.5058444738388062, -0.35041195154190063, -0.6144862771034241, 0.32106897234916687, 1.6443889141082764, 0.7347113490104675, 0.12601523101329803, -1.0392531156539917, 0.6637342572212219, 0.11838482320308685, -0.9635063409805298, -0.4269667863845825, -0.827922523021698, 1.5175671577453613, -0.5440993905067444, -0.4733831286430359, 1.1595927476882935, 1.4641462564468384, 0.387389212846756, -0.07002852112054825, -0.3923927843570709, 1.1742299795150757, -0.12921899557113647, 0.12441155314445496, 0.4280894994735718, 1.1439343690872192, -2.1990017890930176, -1.9313855171203613, -1.3007512092590332, -0.5408768653869629, -0.188572958111763, -0.4477357864379883, 0.10072483122348785, 0.9920734763145447, -0.21839193999767303, 0.08119985461235046, -0.5134326815605164, 1.9810644388198853, -0.6428413391113281, 0.3341660797595978, -0.2790757715702057], [-1.4549189805984497, -1.1256285905838013, -0.009370354004204273, -0.44186466932296753, 0.3016379177570343, -0.7430200576782227, -1.1543782949447632, -0.011060258373618126, 1.7170133590698242, 0.12169286608695984, 0.2533986568450928, 1.0786476135253906, 0.03125365450978279, -0.32540854811668396, 0.5003007650375366, -1.6081547737121582, -0.7164481282234192, 1.7150168418884277, 0.6871266961097717, 0.5216938257217407, 0.06431645900011063, 0.7026134729385376, 0.28292691707611084, 0.7627120018005371, 2.0509891510009766, -1.1243137121200562, -0.5426804423332214, 0.5779951214790344, -1.4915658235549927, -1.121579647064209, 1.4077913761138916, -0.13309313356876373, 0.6911161541938782, -1.0068089962005615, -1.2643145322799683, 0.2541302442550659, -1.0976057052612305, 0.6048778295516968, 0.19257399439811707, -0.4975520372390747, 0.5982038974761963, -0.8629299998283386, 0.24435491859912872, -0.17398416996002197, -1.1687588691711426, 1.3472003936767578, -0.19603587687015533, 1.8385639190673828, 0.0756257027387619, -1.0076478719711304, -0.5035281181335449, 0.9898690581321716, -1.3590283393859863, -1.1117902994155884, 0.4631657004356384, 1.063149094581604, -0.3569880425930023, -0.4155973792076111, 0.6184189319610596, 0.3190285265445709, -0.23509518802165985, -0.2800166606903076, 0.29188504815101624, 0.9399711489677429, -2.242769956588745, -1.7440088987350464, -0.8128805160522461, 0.7538555860519409, -1.479915976524353, 1.061625599861145, -1.3086767196655273, -0.9313240647315979, 0.607093334197998, 0.0653604194521904, -0.5130376815795898, 1.3624969720840454, -0.9331185221672058, -0.2780715823173523, 0.3738894462585449, 1.175091028213501, -0.7469993829727173, 0.026344018056988716, 0.272223562002182, 0.7565047144889832, 1.0109690427780151, -0.974787175655365, 0.6003850698471069, 0.6419731378555298, -0.2530260384082794, -0.14485523104667664, 1.5816466808319092, 1.4388887882232666, 1.0422277450561523, -0.755939781665802, 0.16680945456027985, -0.2530224025249481, -0.0008377619087696075, -0.2851778566837311, 0.606465220451355, -0.35184693336486816, 0.9169138073921204, 1.246271014213562, -0.6995730400085449, -0.4832107126712799, 0.042225394397974014, -0.9194362759590149, -1.7696717977523804, 0.48012733459472656, -1.1058145761489868, 0.10717513412237167, -0.41989830136299133, -0.927497386932373, 1.7690527439117432, 0.4241352081298828, 0.03190242871642113, 0.9298111796379089, 1.0674593448638916, 0.4475841224193573, 0.10776443779468536, -0.20595118403434753, 0.16915248334407806, 0.8117578029632568, 1.2571333646774292, -1.176095724105835, -0.2959970235824585, 0.4071594476699829, -1.2522332668304443, 0.1711837351322174], [-0.5218031406402588, 0.7029337882995605, -0.0924559086561203, -1.8751921653747559, 0.22469480335712433, 0.44833865761756897, -2.1902260780334473, -1.0093272924423218, 1.1088347434997559, 3.0334150791168213, 0.38715776801109314, 0.26063868403434753, -1.9177716970443726, 0.9957942366600037, -1.9517536163330078, -1.3533216714859009, 0.7350893616676331, -0.4993475377559662, -0.0013191510224714875, 0.5842685699462891, -0.06992598623037338, 1.557641625404358, -1.1018600463867188, -1.3700788021087646, -0.6515282392501831, -0.09072330594062805, -0.0947342962026596, 0.5236667394638062, -0.32224705815315247, 0.2911159098148346, -0.42490315437316895, 0.7027931809425354, -0.490786612033844, -0.13865433633327484, 0.24061211943626404, 0.3071759343147278, -0.42324429750442505, -1.311745285987854, -0.22935238480567932, 0.14229334890842438, 0.37996700406074524, 1.847143530845642, -0.6279035806655884, -1.6866756677627563, 0.3840333819389343, 0.9813510775566101, -0.1476452648639679, 1.6376726627349854, -0.477425754070282, -0.10541899502277374, 0.4293186068534851, -0.293729692697525, -1.6291940212249756, 0.8086956739425659, 1.0383104085922241, 1.3840148448944092, -1.6312503814697266, -0.39930227398872375, -0.0244719460606575, 1.1948904991149902, 1.5052438974380493, -1.1596428155899048, -0.28777647018432617, -0.07000721991062164, 0.1003948450088501, 1.2480744123458862, 0.1959535926580429, 0.23663020133972168, -1.6206233501434326, 0.03775635361671448, -0.15945874154567719, -0.4924223721027374, 0.4168892204761505, 0.28565338253974915, -0.8615715503692627, -0.3005092740058899, -1.738120198249817, -0.8494449257850647, 0.06307079643011093, -0.31204190850257874, 1.3630603551864624, 0.050658147782087326, -2.9359052181243896, -0.14797039330005646, 1.1120963096618652, -1.506620168685913, -0.04201788827776909, 1.6695454120635986, -2.4717278480529785, -0.44685637950897217, 0.32188680768013, 2.307281970977783, -0.6958922743797302, 0.3541318476200104, 1.2043012380599976, 1.6803609132766724, 0.8151023983955383, -0.38748443126678467, -0.21126116812229156, 1.6650025844573975, -2.4882655143737793, -0.24721753597259521, -0.6358664035797119, -0.05110330507159233, -0.022110160440206528, -0.29821309447288513, -0.26487669348716736, 0.5212991833686829, 0.22421535849571228, 1.4218817949295044, -1.544042706489563, 0.9202598333358765, -0.95382159948349, 0.8192780017852783, -1.0893782377243042, -0.7868313193321228, -0.7021535634994507, -0.891408383846283, -0.31062328815460205, -0.28475698828697205, 1.9150997400283813, 1.0121196508407593, 0.48441535234451294, -0.3124716877937317, -0.24486792087554932, -0.6113556623458862, 1.3820416927337646, -0.6444088816642761], [-0.4461683928966522, 0.48133689165115356, 0.9994179010391235, 0.005227846559137106, 0.07497463375329971, 0.5359443426132202, 0.4850466251373291, 0.07198154926300049, -1.3300079107284546, 1.3560092449188232, -1.4491077661514282, -0.12240162491798401, 1.3107274770736694, 1.0530930757522583, -1.282724380493164, 1.1959595680236816, -0.4900275766849518, 0.09451381117105484, 0.055689290165901184, 1.055598258972168, -0.24837133288383484, 0.4029950797557831, 1.118203043937683, -2.2679519653320312, 0.8972423672676086, 0.33516231179237366, 0.36888134479522705, 1.3391766548156738, -1.102660059928894, 1.4366333484649658, -1.1999002695083618, -0.39660975337028503, -0.2042863517999649, -1.2695276737213135, 1.0529130697250366, -0.5393415093421936, -1.977135419845581, -0.11312029510736465, -1.2674800157546997, -0.22630009055137634, -1.0497221946716309, 0.3205786347389221, 1.1324962377548218, 0.06610272824764252, 0.5234226584434509, -0.2108222097158432, -0.19496822357177734, 1.6790496110916138, 0.5302493572235107, -0.6446993350982666, 1.0888733863830566, 0.9201539754867554, -0.661228358745575, 1.856224775314331, -1.2196999788284302, -0.597543478012085, 0.11059368401765823, -1.2875163555145264, -0.6128575801849365, 0.9929986596107483, 0.6084112524986267, 0.6476020216941833, -0.5088300704956055, -0.30417802929878235, 0.4560590386390686, -0.13338957726955414, -1.454841136932373, -0.33311381936073303, -0.013907371088862419, 1.0588977336883545, -0.8588242530822754, 0.5698864459991455, -0.7059193849563599, -0.734972357749939, -0.06962230056524277, 2.9117205142974854, -1.544778823852539, -1.7639552354812622, -1.9778594970703125, -0.33308878540992737, 1.1970313787460327, -2.0582079887390137, 2.009566307067871, 0.5664591789245605, -2.1571402549743652, 0.24648813903331757, 1.4883906841278076, 0.06552302837371826, 0.9197423458099365, 0.7379955053329468, 0.5376917719841003, -0.6094271540641785, 1.051606297492981, 0.3235889971256256, -0.14803287386894226, -1.9224324226379395, -1.7515883445739746, 0.21542705595493317, -0.7049407362937927, 0.06960161030292511, -0.10050518810749054, -1.6819742918014526, -1.4259588718414307, 0.030590977519750595, -1.1625337600708008, -0.2241659015417099, 0.08726520836353302, 1.1021924018859863, 0.16064453125, 1.0405739545822144, 0.08517450094223022, 0.057995300740003586, -0.49616754055023193, 1.7012393474578857, -0.508965253829956, 0.2146279215812683, -1.3040400743484497, 0.4424978196620941, 0.98827064037323, 0.18801121413707733, 0.4802938997745514, 1.0411604642868042, 0.8226026892662048, -1.2351006269454956, -0.5061911344528198, -0.21877501904964447, 0.16736307740211487, -0.8587132096290588], [-1.6709251403808594, 1.6397705078125, -0.6160966157913208, -0.6684243083000183, 1.0285311937332153, -1.9787218570709229, 0.7870346307754517, -1.053165316581726, 0.5243290066719055, 0.410550057888031, -1.8644813299179077, 0.2565722167491913, 1.7558835744857788, -0.34912851452827454, 0.07699575275182724, -0.41484537720680237, 0.05218590423464775, 1.0829583406448364, 1.0040515661239624, -0.6516132354736328, -0.747721254825592, -1.521407961845398, 0.5369576811790466, -0.41190385818481445, -2.2174344062805176, -1.2928948402404785, 1.7719732522964478, -1.3228850364685059, -0.8926188945770264, -1.0127732753753662, -0.9031786918640137, 0.6930099129676819, 0.9914812445640564, 0.897021472454071, -0.03851349279284477, 0.17792202532291412, -0.4435156285762787, -0.23105449974536896, -0.6354948282241821, -0.37275347113609314, -0.7976747751235962, -1.2658089399337769, -0.36884254217147827, -1.7059903144836426, 0.2958550751209259, -3.00527024269104, 0.25202131271362305, 0.33094391226768494, 0.2785080373287201, -0.06011855974793434, -0.8610068559646606, -1.694321632385254, 0.28319838643074036, 1.615209698677063, -1.5996549129486084, 0.4742780327796936, 0.22115297615528107, -1.0611029863357544, -0.5386652946472168, -0.07959683239459991, 0.7740404009819031, 1.7925175428390503, 1.826237678527832, -0.7363994121551514, 0.2903333902359009, -0.26098981499671936, -0.45545250177383423, -1.2519221305847168, 0.5269181132316589, -1.5242464542388916, 1.1406800746917725, -0.3240857720375061, -0.10446605086326599, 0.3098739683628082, -0.5650490522384644, -0.2679286599159241, -0.6409469842910767, 0.7543034553527832, 1.192604660987854, 0.3604643940925598, -0.06690935045480728, 0.3660551607608795, -0.905119776725769, -0.029951859265565872, 1.0934031009674072, 0.06975952535867691, 0.46098437905311584, -2.2840464115142822, -0.13483703136444092, -0.35484224557876587, 0.8415235877037048, -0.9110575318336487, 2.2951600551605225, 0.34241506457328796, -0.435656875371933, 1.0884931087493896, -0.27807798981666565, 0.5150513052940369, -1.1932857036590576, -0.16614685952663422, 1.8326780796051025, 0.4279382526874542, 1.2393521070480347, 0.7207130789756775, -0.38994935154914856, -1.6412544250488281, 0.28155970573425293, 1.380718469619751, -0.5487604141235352, 1.146636724472046, 0.9992477297782898, 0.43610161542892456, -0.2878299653530121, 0.5513497591018677, 0.5535387992858887, 2.0862650871276855, 1.6172466278076172, -1.093319296836853, 0.6103062629699707, -1.6978156566619873, -0.5227369070053101, 0.25456807017326355, -1.0942018032073975, -0.5259488821029663, 0.009124789386987686, 0.3185466527938843, 0.23669768869876862, 1.063256025314331], [1.2307952642440796, 0.13517709076404572, -1.1334068775177002, -0.37130501866340637, -1.1727076768875122, -0.6816905736923218, 0.6922377347946167, 0.617904782295227, -0.7105311155319214, -0.3235861361026764, 0.47742223739624023, -0.23642529547214508, 0.1353342980146408, -0.7348877787590027, 0.5966951847076416, 0.07016590982675552, 2.1031272411346436, 0.025654690340161324, 0.4435275197029114, 1.6156296730041504, 0.6281968951225281, 0.9811094999313354, 0.6571848392486572, 0.7424235939979553, -0.5296711921691895, -0.3360752761363983, -0.038690902292728424, 0.7168195247650146, 1.0565358400344849, -0.08613140136003494, 1.3303043842315674, 1.216676115989685, -0.9349687695503235, -0.1842900961637497, -0.11371252685785294, 1.166882038116455, -0.02808755822479725, -0.30811265110969543, 0.7611427903175354, 0.5137409567832947, -0.1495320349931717, 1.260188341140747, -1.2395517826080322, -1.7313796281814575, -0.26182156801223755, 2.188624143600464, -2.2084524631500244, -0.4721459746360779, -0.7837260365486145, -1.7643688917160034, -0.17129193246364594, 1.0852735042572021, -1.9077085256576538, 1.5323189496994019, -0.8061941862106323, -0.042092107236385345, 0.6761424541473389, -1.4520926475524902, -0.02711988054215908, -1.2735923528671265, 0.25231650471687317, 0.29952070116996765, -0.02575622871518135, 0.09034159779548645, -1.2560551166534424, 1.582293152809143, 0.42694151401519775, -0.18413542211055756, -0.16641725599765778, 0.43185245990753174, 1.1443486213684082, 1.572059154510498, -0.9718773365020752, -1.3364707231521606, -0.34704530239105225, -0.930569052696228, -2.1685290336608887, 0.49530553817749023, -0.5054472088813782, 0.6386947631835938, -0.2807788550853729, -0.6375459432601929, -0.7460227608680725, -0.7209078073501587, -0.025467809289693832, 0.04489052668213844, 2.4602103233337402, 1.048534870147705, -0.3770066201686859, 0.8096867203712463, -0.1531146615743637, -0.9788652062416077, 0.24658751487731934, -1.6792192459106445, 2.928609609603882, 1.140606164932251, -0.7159045338630676, 0.3979716897010803, -1.23239266872406, -0.3927546441555023, 1.0430649518966675, 1.1039377450942993, -0.9606306552886963, -0.8153195977210999, 0.40222638845443726, -1.6600487232208252, -0.2280939370393753, -1.2994939088821411, 0.47265589237213135, 1.1844242811203003, -2.5407862663269043, -1.065382719039917, -0.04921959340572357, 2.107060194015503, 1.5318423509597778, -0.5958754420280457, -0.6064426898956299, 0.09571082144975662, 0.3906711935997009, 2.8598015308380127, 1.5677541494369507, 0.13541576266288757, -0.797924280166626, -1.0184534788131714, 0.4927235543727875, -1.3303394317626953, -0.21661685407161713, 1.045657753944397], [-1.3136625289916992, 0.5410088300704956, 0.6989359855651855, 1.9146158695220947, 0.9674168825149536, -0.549312949180603, -0.1627204269170761, 0.6883724331855774, -0.01648438163101673, -0.5235732197761536, 0.8860735893249512, 0.8871996402740479, 0.7350234389305115, -0.051340702921152115, 0.5437618494033813, -0.7358782887458801, -0.14399750530719757, 1.2767632007598877, 0.2792837917804718, 0.03966839984059334, -0.27165448665618896, 0.8198856711387634, -1.3895771503448486, 0.27277034521102905, 0.22878213226795197, 1.0805259943008423, -0.5887590646743774, 0.8311144709587097, 0.7746235132217407, -1.2266603708267212, -0.14143206179141998, 0.07840505987405777, 0.21460849046707153, -0.6506487131118774, -0.4215969443321228, 0.5237174034118652, -0.46121692657470703, -0.7299726605415344, 0.32585233449935913, -0.3155786693096161, -1.4919489622116089, 0.35308384895324707, 2.244600296020508, 1.9256647825241089, 1.3720788955688477, 1.624679684638977, -0.1776675134897232, 0.6982225179672241, -0.9410890936851501, -0.3358718752861023, -1.4466242790222168, -0.3920212388038635, -1.8536313772201538, 0.40434542298316956, 0.33730292320251465, -0.6861308217048645, -1.3748674392700195, 1.9168177843093872, -0.9366881847381592, 0.19578586518764496, 1.448167085647583, -1.1086821556091309, -0.05083290860056877, 0.8468475341796875, 1.278215765953064, 2.029949426651001, -1.18468177318573, -0.23677456378936768, -1.0920300483703613, 1.2225096225738525, 0.506441593170166, 1.0382989645004272, 0.03498884662985802, -1.286727786064148, -0.8114867210388184, -0.5148258209228516, -1.7737351655960083, 0.26540887355804443, -0.09987278282642365, -0.02292739227414131, -0.9691578149795532, 0.14312537014484406, -0.010365457274019718, 0.4848315417766571, -0.8852823376655579, -1.4248510599136353, -0.573823869228363, 1.1597402095794678, -0.9448297023773193, 0.5256754755973816, -0.6238406300544739, -0.6985812783241272, -0.7658742070198059, 0.11812643706798553, -0.4629691541194916, -1.067426085472107, 2.0234036445617676, 1.5631792545318604, 1.053248405456543, -0.6178939938545227, -0.1337282806634903, 2.438591718673706, 0.32413890957832336, -1.344443440437317, 0.5345983505249023, 0.5256394743919373, -0.5765377283096313, 0.0754573792219162, -0.3977574408054352, 0.722846269607544, -0.7920037508010864, 0.3717104196548462, -0.6426270604133606, -0.5344972014427185, 0.9738653302192688, 1.2633901834487915, -1.675162672996521, 1.173163890838623, 0.10604798048734665, 0.9906370639801025, -0.7397177219390869, -0.11791294068098068, -0.6221038103103638, -0.15281352400779724, 0.15521498024463654, 1.7357937097549438, 0.05861053615808487, -1.122954249382019], [0.0919249951839447, 1.5785726308822632, 0.016953187063336372, -0.5744251608848572, -1.0151667594909668, -0.27755770087242126, -0.6193227171897888, 0.40666431188583374, -1.4573309421539307, 1.131597638130188, -0.19021472334861755, 0.7514354586601257, 1.1360899209976196, 1.253313422203064, 0.6468419432640076, 0.8845764994621277, 0.34633421897888184, 1.6095682382583618, -0.7853432297706604, -0.20140720903873444, -0.2106168568134308, 0.3183762729167938, 0.9834615588188171, -1.033130407333374, -1.6522157192230225, 0.8806320428848267, -0.5958388447761536, -0.987492561340332, 0.43761566281318665, -1.018431544303894, 1.6127595901489258, 0.16771860420703888, -0.4530109167098999, 0.8623137474060059, -2.1692848205566406, 0.08818525075912476, 1.472852349281311, -0.5335259437561035, 0.791862964630127, 1.0574641227722168, 0.6414585113525391, -0.7817811965942383, 0.9603762030601501, 1.0347275733947754, -1.1128120422363281, -0.25250735878944397, -0.44560423493385315, -1.2810560464859009, 0.6732425689697266, 0.20464615523815155, -1.6659656763076782, -1.9986579418182373, -0.052361950278282166, 0.15335793793201447, 0.3806452751159668, 0.08217493444681168, -0.27194878458976746, -1.6026828289031982, 0.40187695622444153, 1.1624058485031128, -0.06192764639854431, -0.8470535278320312, -1.3052852153778076, -0.10401541739702225, -0.34500253200531006, 1.2896783351898193, 0.12012308835983276, 0.15836048126220703, -0.47504961490631104, 0.2881332039833069, 0.06940055638551712, 0.9001127481460571, -0.2041868418455124, -1.6313310861587524, 1.1428298950195312, 0.6657795906066895, 0.9807119965553284, 0.7866405248641968, -1.6236733198165894, 2.211556911468506, 1.976629376411438, -0.2727157473564148, -0.7592760324478149, -0.6624000072479248, -0.6832334995269775, -0.4535791873931885, 0.9996987581253052, 0.6068860292434692, 0.4745025038719177, 0.9437766671180725, 0.6034359931945801, -2.106691837310791, 1.0094873905181885, 0.3334319591522217, -1.1552499532699585, 0.3423328101634979, 1.1456972360610962, 0.7450716495513916, 0.07469990849494934, 2.470785617828369, -1.6844520568847656, 0.0755573958158493, -0.8256655335426331, -0.7739307284355164, 0.5405124425888062, -0.4783138036727905, -0.704078733921051, -0.7356189489364624, -1.7425682544708252, 0.09030493348836899, -1.2054951190948486, 0.9063469767570496, -1.7103244066238403, 1.3917678594589233, 2.9122581481933594, -0.6143805384635925, -0.7552535533905029, -0.7194985151290894, -0.7057989835739136, 0.5754778385162354, -0.16676995158195496, -1.7076520919799805, 0.22697225213050842, 0.27867770195007324, 0.21714018285274506, -0.5652585029602051, 0.01765880547463894, 0.11410467326641083], [1.5154087543487549, 0.3192194104194641, -0.07480264455080032, -1.078963279724121, -1.4411712884902954, 1.4114117622375488, -0.4379889965057373, 1.5907368659973145, -0.6968638300895691, 1.6697684526443481, 1.2527422904968262, -0.45125913619995117, 1.4075077772140503, 0.8143274188041687, 0.45568081736564636, 0.024360444396734238, -0.04420242831110954, 1.3526897430419922, 1.5289714336395264, 0.4374496340751648, -0.5386880040168762, 2.620389223098755, -1.9292361736297607, -2.414815664291382, 0.7526798844337463, 0.7857122421264648, 0.8817675709724426, -0.22947314381599426, 1.2053122520446777, 1.3492308855056763, 0.7250182032585144, 0.9066879153251648, -0.07346992939710617, 1.0685590505599976, 0.5639780759811401, 0.25220683217048645, -0.14555723965168, -0.500480592250824, -0.7724088430404663, -1.0824049711227417, -0.03580267354846001, -1.2569252252578735, 0.22347940504550934, 0.07871677726507187, 0.00708010932430625, 0.16308631002902985, 0.5278992056846619, -0.07051099091768265, 0.4299173355102539, 0.020855726674199104, -0.6561575531959534, -0.02964097075164318, 0.2406478375196457, -1.3026175498962402, 0.18927356600761414, 0.48959940671920776, -1.5531375408172607, 1.5089032649993896, -0.13675053417682648, -0.6321784257888794, 0.6177763342857361, -0.0558488667011261, -0.7238211035728455, -1.8658150434494019, 0.5537014603614807, 1.7625797986984253, 0.31297144293785095, -0.9140850901603699, 0.2442699670791626, -1.5211416482925415, -0.1976393461227417, -0.6967612504959106, 0.22857271134853363, -0.822930097579956, 0.5975597500801086, 0.4973660111427307, 1.0348328351974487, -1.1503607034683228, 2.597313165664673, 1.5025917291641235, 0.025064710527658463, 1.8230679035186768, 0.36997362971305847, 0.9379059076309204, -1.0128967761993408, 0.4911002218723297, 0.548396110534668, -0.23744039237499237, 0.253487765789032, -0.05830216780304909, 0.7763683199882507, 0.9108672142028809, -0.9412415623664856, -0.010324000380933285, -1.351486086845398, -2.7280352115631104, -0.16394855082035065, -0.6928411722183228, 1.4863581657409668, -1.8700796365737915, -0.20428244769573212, -0.612625777721405, 0.41429227590560913, 0.6657811999320984, 0.3788726329803467, -2.5325145721435547, 0.27978241443634033, -0.32164543867111206, 0.8266626596450806, 0.5632309913635254, -0.6375461220741272, -1.704068660736084, 0.1807529181241989, -0.8338614702224731, 1.396114706993103, -1.4455971717834473, -0.1259499341249466, -2.2952263355255127, -0.1654852330684662, 0.22476741671562195, -0.5008773803710938, 0.1935606300830841, -0.8007928133010864, 0.7563053965568542, 0.12095217406749725, 0.0794520229101181, 1.1790673732757568, 0.2883424162864685], [-1.0103849172592163, 1.441996693611145, -1.6855729818344116, -0.47689178586006165, 0.8293086290359497, 0.6795175075531006, -1.139176845550537, -1.9411641359329224, 1.5408607721328735, -1.5801453590393066, -1.106080412864685, 0.09812891483306885, -1.3233444690704346, -0.04220627620816231, -0.5420350432395935, -0.03718838468194008, -0.9564113020896912, 0.9151192307472229, -0.43564194440841675, 0.06493586301803589, 0.6111485362052917, 0.2929941713809967, -1.173938512802124, -0.6253952980041504, 1.2404553890228271, 0.05883410945534706, -0.14489808678627014, -1.0955654382705688, -2.3417913913726807, 1.0521538257598877, 1.2567579746246338, -0.45788875222206116, 0.6989341378211975, 1.6496293544769287, 0.4687209725379944, 0.11669591814279556, -0.28408893942832947, 0.11210179328918457, -0.6120401620864868, 0.5458862781524658, 0.01811082661151886, 0.7600155472755432, -0.3505917191505432, 1.2752894163131714, -0.774315595626831, -0.43851783871650696, 1.2605557441711426, -2.2421071529388428, 0.6975842118263245, -0.3683820366859436, -0.1257418543100357, -0.005422830116003752, -0.14644740521907806, -0.6096228957176208, -0.2819939851760864, 0.7027990818023682, -0.06861531734466553, -0.3150642216205597, 0.8169461488723755, 1.423809289932251, -0.09777780622243881, 0.4240037202835083, 0.3186601400375366, 1.0262937545776367, 0.7250328063964844, -0.6396105885505676, 0.5649417638778687, -0.5542802214622498, -0.2778129577636719, -0.642840564250946, -0.2768184244632721, 0.8210796117782593, -1.095991611480713, -0.13141393661499023, 0.7970859408378601, -0.6582493782043457, 1.1357768774032593, -0.3334105312824249, 0.3446728587150574, 0.17069052159786224, 0.3317885100841522, 0.8688930869102478, 0.32540062069892883, 0.761480450630188, -2.1747329235076904, 0.6730872392654419, 0.09702306240797043, -0.44184255599975586, 0.48887351155281067, -1.2106573581695557, 0.8328756093978882, -0.8786812424659729, -0.12802371382713318, 0.4571654200553894, 0.2054266482591629, -1.083142638206482, -0.701538622379303, -0.9542118906974792, 0.40140780806541443, -1.121822714805603, -1.8821707963943481, -1.4128273725509644, 0.14368122816085815, 0.6711599230766296, -0.5763000845909119, -1.1035528182983398, -0.5107119083404541, -1.5334711074829102, 1.6477676630020142, 1.929569125175476, 0.6495839953422546, -0.05590282753109932, -0.173019677400589, -1.003310203552246, -0.19324098527431488, 0.36316341161727905, 1.3260024785995483, -0.4158453047275543, 0.5871168971061707, -1.608381986618042, 0.1954469233751297, 1.1695897579193115, -0.7795612812042236, -1.2142012119293213, 0.6782609224319458, -0.49885380268096924, 1.3045600652694702, 0.5915175676345825], [-0.24428583681583405, -1.3994156122207642, -0.21229447424411774, -0.4056183397769928, 0.7297934889793396, -1.201417326927185, -2.050187110900879, 0.5890326499938965, -1.2286670207977295, 1.0547226667404175, 0.42403119802474976, -1.2375701665878296, 0.26945117115974426, 1.0233017206192017, -0.8127988576889038, -0.8942549824714661, -0.8188967704772949, 0.9723465442657471, -1.8316086530685425, 0.537536084651947, -0.1412510871887207, 2.5795655250549316, 1.0986918210983276, 1.1711328029632568, 0.9339225888252258, 0.35498109459877014, -0.1998177021741867, 0.9659168720245361, -1.0602681636810303, 1.4136639833450317, 1.5681334733963013, -1.5184351205825806, 0.5049853920936584, 0.03888707607984543, 1.6081725358963013, 0.1213742047548294, -0.12126680463552475, 0.9371384382247925, -1.288477897644043, 0.7715803980827332, -0.25906267762184143, -0.5806604027748108, -1.4530800580978394, -1.5365211963653564, 0.9598013758659363, 0.8798685669898987, -1.0897077322006226, -0.6468079090118408, 1.5488287210464478, -0.8395880460739136, -0.4385188817977905, -1.7180542945861816, 0.5788845419883728, 1.4737544059753418, -0.5401154160499573, -0.747815728187561, 0.361653596162796, 0.8009774088859558, -0.36481261253356934, 0.16136547923088074, 0.8569849133491516, -1.1616463661193848, -0.6647651195526123, 1.5980992317199707, -2.5084609985351562, 0.27387163043022156, -1.0837268829345703, 0.17106205224990845, -0.8951189517974854, 1.389543890953064, -0.45460665225982666, -0.6050605177879333, 1.3977837562561035, -0.33532923460006714, 1.2977792024612427, -0.3460792005062103, 0.1506214737892151, 0.9173771739006042, 0.45681530237197876, -0.6904842853546143, -2.2989957332611084, 0.009767808951437473, -0.5158407092094421, 0.22206152975559235, -0.36696845293045044, -1.9426337480545044, -0.2092273086309433, 0.7245169281959534, -0.16596387326717377, 0.29224085807800293, -1.9145361185073853, -2.2628231048583984, -1.194940447807312, -1.0120327472686768, -0.7009164690971375, 1.878187894821167, 1.6147533655166626, -0.6354615688323975, 0.513319194316864, 0.7911570072174072, -0.4675188660621643, 0.5058299899101257, -0.3912445604801178, -0.7496197819709778, 0.24260544776916504, -1.371777892112732, -0.4709748923778534, 0.3128156065940857, -0.05844428762793541, 0.06953680515289307, 0.35302841663360596, 2.2149267196655273, 0.985759973526001, -1.3756344318389893, 0.6913616061210632, 0.8413196802139282, -0.4940088987350464, -0.30709004402160645, 1.5968090295791626, -1.8727890253067017, -0.8115977048873901, -0.8091955184936523, 1.149231195449829, -0.5041818022727966, -0.37824881076812744, -0.016944680362939835, -1.2107291221618652, -0.7967907190322876], [-0.7132796049118042, 0.5050526261329651, -0.5382265448570251, -0.16337881982326508, 0.04390009492635727, 0.368216872215271, 0.7990867495536804, -0.12614087760448456, -0.2115650475025177, 1.3222404718399048, -0.18414533138275146, -0.8262753486633301, 0.0395328551530838, -0.3034154176712036, 0.19112460315227509, -0.3493344783782959, 0.17511184513568878, 0.6452386379241943, 0.06973132491111755, -0.40176859498023987, -1.11644446849823, -1.3150144815444946, -2.496927499771118, 0.09145786613225937, 0.532892107963562, -0.9125725626945496, 0.8885149359703064, 0.5200193524360657, 0.31742122769355774, 0.05626463145017624, -0.39549580216407776, 1.52711820602417, -0.06378908455371857, 0.5889162421226501, -1.3372044563293457, 0.7315640449523926, -0.39738842844963074, 1.535064935684204, -0.43030402064323425, 0.7680112719535828, -0.05485450103878975, 0.95048987865448, -0.5377920269966125, -1.568776249885559, 0.7681140303611755, 0.6617013812065125, 0.8425474166870117, -0.21496260166168213, 0.4801144599914551, -0.55108243227005, -0.06454722583293915, -2.0603694915771484, -1.6050097942352295, 1.893703579902649, 0.4578307867050171, 1.9694758653640747, -1.5677554607391357, 0.5187757611274719, 1.6010618209838867, 0.3447434902191162, -0.4004327356815338, -1.159131646156311, 1.042615294456482, -0.2842102646827698, -1.5165022611618042, -0.05490608885884285, 0.18580852448940277, 0.35219788551330566, -1.7670814990997314, -1.0235910415649414, 0.02623808942735195, 0.9261633157730103, -1.1892178058624268, 1.7582789659500122, 0.034646596759557724, -0.6002342104911804, -1.000632882118225, -0.21671149134635925, 0.3608051538467407, 0.21232351660728455, -1.0577701330184937, -0.4931659400463104, 0.07340999692678452, 1.7543312311172485, -0.4733564853668213, -1.732055902481079, 1.1183918714523315, 1.8658190965652466, -0.34391704201698303, -0.7989981174468994, 0.15837322175502777, -0.08146501332521439, -0.38783735036849976, -1.497812271118164, -1.039544939994812, 1.1738404035568237, -0.15289415419101715, -0.572913646697998, 0.9845051765441895, 1.3033171892166138, -1.4096338748931885, 0.7086793184280396, -0.690085232257843, 0.38710692524909973, -1.4631402492523193, -0.4258081912994385, 0.8984795808792114, -0.4292192757129669, -0.21201223134994507, 0.5039932727813721, 1.1385619640350342, -0.6150403618812561, -0.28459277749061584, 0.5206075310707092, -0.3196326792240143, 0.040705688297748566, -0.9661145806312561, -1.117978572845459, 1.1328492164611816, 0.3839842677116394, -0.07267850637435913, -1.035636067390442, 0.4216795563697815, -0.21814721822738647, -1.430862545967102, -0.8272383809089661, -0.8192533254623413, 0.17785508930683136], [-0.9486106038093567, -0.2989146113395691, -0.1518661230802536, -0.4844020903110504, -0.6231810450553894, -0.8398474454879761, 0.11700178682804108, 0.13197766244411469, -1.2096816301345825, 0.29927849769592285, -0.6855964660644531, -0.1524457186460495, 2.1678011417388916, 1.1215875148773193, 0.8056054711341858, -0.6217777729034424, 0.1469760537147522, -0.07127255946397781, -0.3980462849140167, -0.45641404390335083, -0.651441752910614, 1.1209262609481812, 0.8992622494697571, 0.5138495564460754, 0.46182388067245483, -0.9039218425750732, 0.6714590787887573, -1.517673134803772, 0.7584510445594788, -1.4741204977035522, 0.05581391602754593, -0.8430885672569275, -0.13777028024196625, 1.0785295963287354, 0.2896682024002075, -1.211863398551941, 1.8629217147827148, -0.29287317395210266, 1.2215300798416138, -1.1994078159332275, -0.19910559058189392, 0.8354569673538208, -0.12647688388824463, 1.6511237621307373, 2.4890286922454834, 0.43650341033935547, -0.6250463724136353, -0.5730199813842773, -0.33255094289779663, 0.7357404828071594, -0.8318642973899841, -0.4889245927333832, 0.28155601024627686, -0.34906917810440063, -0.14997483789920807, 0.027362024411559105, -0.8437819480895996, 1.1413112878799438, 0.027860749512910843, -1.3435171842575073, 0.5666773319244385, -0.5648674368858337, -0.1408453732728958, 0.2557504177093506, 0.25680986046791077, 0.5590620636940002, 0.1988188624382019, 0.15141203999519348, 0.3922370970249176, -0.05992758646607399, -1.52760910987854, -0.05692819505929947, -2.2692551612854004, -0.38624557852745056, -0.20212583243846893, 0.39677688479423523, 1.7714730501174927, 0.46974098682403564, -1.0292463302612305, 1.1397502422332764, -0.8403555750846863, 0.5543559193611145, 1.0226446390151978, 0.38989171385765076, 0.46037358045578003, 0.002408513566479087, 0.17795543372631073, 1.2207111120224, 0.5170705318450928, 1.049392580986023, 1.1455751657485962, 0.06634336709976196, 0.5145936012268066, 0.6610761284828186, -0.44939979910850525, 0.46738144755363464, 1.1480597257614136, 1.2484245300292969, -0.11754801124334335, 0.04711485281586647, -0.7185596227645874, 0.23783724009990692, -1.0914852619171143, 0.31372833251953125, 0.2007957398891449, -2.208019495010376, -0.6491191983222961, -1.524577260017395, -1.5779573917388916, -1.064101219177246, -0.6916552782058716, 0.5990832448005676, -0.8223875164985657, -0.12319561839103699, -0.8879932165145874, -1.003321647644043, 2.1006035804748535, -1.1833738088607788, 0.7681323289871216, -0.27292224764823914, -0.6879467368125916, -0.7823575139045715, 1.1985524892807007, 0.33782926201820374, -1.0903927087783813, -0.8668261766433716, -1.044975757598877, 0.5199653506278992]], [[0.8898592591285706, 0.11592521518468857, -0.24902048707008362, -0.06277771294116974, -0.684093713760376, -0.6251939535140991, 0.17621320486068726, -0.8610816597938538, -1.2999659776687622, -1.0224666595458984, 0.2106967270374298, -1.0782761573791504, -0.11196643114089966, -1.0059738159179688, -1.485027551651001, -0.6205669045448303, 1.9722548723220825, -0.016081677749753, 0.7846934795379639, -1.5698330402374268, -1.090921401977539, 0.5852490067481995, 1.416765570640564, -2.0972416400909424, -1.0948435068130493, -0.3932490944862366, 2.050558090209961, 0.8721380829811096, -1.749694585800171, 0.5432680249214172, 0.3710459768772125, 0.6168151497840881, -0.35596367716789246, 1.5637226104736328, 0.8059526681900024, -2.341721534729004, 1.206620454788208, 1.002734661102295, -1.6147710084915161, 0.12540803849697113, 0.7984763979911804, 0.8419528603553772, 0.12044163793325424, 1.1157407760620117, 0.35930904746055603, 1.8799363374710083, -0.7906107306480408, 0.5284229516983032, -1.2294597625732422, 0.8622264266014099, -0.34023115038871765, 1.1307669878005981, -0.6745755076408386, 0.35861456394195557, -0.28630420565605164, -1.2165552377700806, 0.2011740505695343, 0.8955813050270081, 0.43421056866645813, 0.8280060291290283, -1.4350199699401855, -0.447116494178772, -0.23130619525909424, 0.4717118442058563, 1.406071662902832, 1.0862864255905151, -1.3265670537948608, -3.154057264328003, -1.7336610555648804, -1.9629288911819458, 0.7089514136314392, 0.5027465224266052, 1.636680245399475, 0.13595977425575256, -2.06424880027771, 1.4409942626953125, 0.8518317341804504, -0.7195355892181396, 0.6629471778869629, -0.5785166025161743, -0.35686907172203064, -0.3552856147289276, 1.4720507860183716, -1.2931251525878906, 1.4394958019256592, 0.581195592880249, 1.1896469593048096, -1.851188063621521, -0.16912133991718292, 0.49017333984375, 0.3303217589855194, 0.23597054183483124, -1.2094018459320068, 0.10918310284614563, 0.3905787467956543, 0.005636183079332113, -0.2934451699256897, -0.4288868308067322, -0.294382780790329, -0.4996834099292755, 0.9555712342262268, -1.3768614530563354, -0.007832442410290241, -0.5802300572395325, 0.7616788148880005, 0.5213080644607544, -0.1362404078245163, 2.0970938205718994, 0.14996542036533356, -0.6773099303245544, 1.520397424697876, -0.6400196552276611, -1.269835352897644, 0.3574499189853668, 1.6528595685958862, 0.4218524992465973, -0.17785270512104034, 0.7044246792793274, 1.0093036890029907, 1.075732946395874, 0.12431564927101135, 0.03455961495637894, -0.10925588756799698, -1.5102113485336304, -0.8276099562644958, 0.6258722543716431, -1.216320276260376, -0.694000244140625], [-0.4056631028652191, 1.1529245376586914, -0.37117528915405273, -1.3759697675704956, 0.13895642757415771, -1.2484484910964966, -0.10280536860227585, -0.057424310594797134, -0.23811687529087067, 0.9698678851127625, 1.6468995809555054, -0.7327072024345398, -0.06900745630264282, -0.4261666238307953, 0.04247170314192772, -0.826161801815033, -0.16017599403858185, 2.7740390300750732, -0.3413236141204834, -0.503217875957489, -1.7071950435638428, -0.728168249130249, 0.9056350588798523, 1.3901845216751099, -0.6708559393882751, -0.42187872529029846, -1.644471526145935, 0.6888418793678284, 1.8614482879638672, 1.0246325731277466, -0.7260721921920776, -0.2816004455089569, 1.9049819707870483, -0.12729208171367645, -1.0479556322097778, -0.09746651351451874, 0.18606777489185333, 0.850608766078949, -0.4353177547454834, -0.733173131942749, -0.41898611187934875, -0.7555115818977356, 1.3014841079711914, 0.7699338793754578, -0.4797736406326294, 1.9086402654647827, 0.17900162935256958, -0.7208360433578491, -1.1130354404449463, -0.7479997277259827, -0.6790528893470764, 1.947982668876648, 0.26179268956184387, 1.079663872718811, -1.134002447128296, 1.8128514289855957, 0.8936378955841064, -0.5232741236686707, -0.022859910503029823, 0.6304248571395874, 0.5254692435264587, 0.2967170178890228, -0.9463772773742676, -0.6555909514427185, -0.5814933776855469, 0.3245108425617218, -1.3059359788894653, 0.0669291689991951, 0.2507278025150299, -0.8707872033119202, 0.2751263380050659, 1.1851588487625122, -1.2233421802520752, -0.2619841992855072, -0.23479127883911133, 1.7341872453689575, -0.6926006078720093, 0.13236358761787415, 0.751370370388031, 0.045162566006183624, -0.22087037563323975, -1.2140322923660278, 1.5323066711425781, -0.9100985527038574, -0.6908209919929504, -0.8079398274421692, -0.20826996862888336, 0.35033902525901794, -1.2531050443649292, -0.6273778676986694, 0.4271191656589508, -0.15737107396125793, 0.016367273405194283, 1.2037500143051147, 0.9844134449958801, -2.0856916904449463, 1.0068279504776, 1.1788277626037598, 0.9783933162689209, 1.5539188385009766, -0.26878225803375244, 1.0311187505722046, -0.42806291580200195, 1.2593172788619995, 1.6822631359100342, 1.5106781721115112, 0.41297608613967896, 2.6024856567382812, -0.9864934086799622, 0.9153038859367371, -1.0550446510314941, 0.513306200504303, 1.6452679634094238, -0.43560466170310974, -0.7381916642189026, 2.906200647354126, 0.7925059199333191, -1.8445155620574951, 0.37241485714912415, -0.8578010201454163, -1.585360050201416, 0.8583605289459229, 0.32467278838157654, 0.11999693512916565, 1.8727024793624878, -1.6691805124282837, -1.0710887908935547, 1.2327191829681396], [0.09501849859952927, 0.4488329291343689, 2.196078300476074, 0.5531048774719238, 1.3939343690872192, -0.2840711772441864, 0.15046019852161407, -0.6299558877944946, 1.623311996459961, -0.19638149440288544, 0.8630428314208984, 0.6270314455032349, 1.1834441423416138, -1.5202778577804565, -0.7707908749580383, -0.19575391709804535, -0.8355717658996582, 0.16375823318958282, -0.17249098420143127, 0.5919408202171326, -0.7504158020019531, 1.442728877067566, -0.05584347993135452, -0.42351633310317993, -0.8275933265686035, 0.4197070896625519, -1.8737072944641113, -0.8317362070083618, 0.14731024205684662, 0.441739946603775, 0.016958650201559067, 0.4327848553657532, -0.7865591049194336, 2.1532583236694336, 0.5299566388130188, 0.7641536593437195, 1.678269863128662, -0.16982047259807587, 0.511989176273346, -0.5054499506950378, 0.2256269007921219, 0.0997229740023613, 0.11747045814990997, -1.1717793941497803, 0.280122846364975, -0.17907072603702545, 0.7553830146789551, -1.0953214168548584, 0.3638654947280884, 1.396120548248291, -0.43627843260765076, 0.7454147934913635, -0.5308341383934021, -0.8840515613555908, 0.3649204969406128, 0.966407299041748, -1.2010047435760498, -0.8434270024299622, 0.450111985206604, 0.028792478144168854, -0.213418111205101, -1.1923043727874756, 0.0244553554803133, -1.8060060739517212, 0.6779887676239014, 0.5498155355453491, 0.5770070552825928, 1.4885603189468384, 0.2608475983142853, 0.5957857966423035, -1.2183685302734375, -0.6395339965820312, 0.7507774233818054, -0.11532702296972275, 0.3013802468776703, -0.3375069200992584, -0.8279799818992615, -0.4534488022327423, 1.072898268699646, 0.9111570715904236, 1.0543593168258667, -1.60887610912323, -0.9144587516784668, 0.057383447885513306, 0.5139980316162109, 0.47818952798843384, -1.6568171977996826, -0.12997369468212128, 0.008968422189354897, -0.144250750541687, 0.9666402339935303, 0.8788735866546631, -1.3698773384094238, 0.14148841798305511, -0.13346119225025177, -2.456557512283325, -0.48842352628707886, -2.5854666233062744, -0.415323406457901, 0.5021994709968567, 0.2326478511095047, 1.4780323505401611, -0.3830656111240387, -0.3003598153591156, 0.47294557094573975, -0.35583704710006714, 0.24002134799957275, -0.041277844458818436, -0.45961859822273254, 0.4651612639427185, -0.7381296753883362, -1.1164017915725708, 1.1792926788330078, 0.6849815845489502, -1.4910520315170288, 0.4192158579826355, -1.3443372249603271, -0.6419165134429932, -0.33448857069015503, -1.388564944267273, 1.6073980331420898, -0.9527348875999451, 0.5582260489463806, 1.730054497718811, 1.2917819023132324, -1.3336361646652222, 0.10626357048749924, 0.10055246949195862], [1.2009174823760986, -0.8675821423530579, 0.5761470794677734, -1.5438376665115356, 0.13041968643665314, 1.1756283044815063, -0.07015308737754822, -0.9712795615196228, -0.5581849217414856, 1.2248320579528809, -1.6253644227981567, 0.9138697385787964, -0.5298187732696533, 0.7297332286834717, 0.6150838732719421, 0.02559727616608143, -1.2697839736938477, -0.9979952573776245, 1.8046444654464722, -0.9193978309631348, -0.15768858790397644, -0.6793801784515381, 3.429375171661377, -0.4819544553756714, 0.08249883353710175, -0.08002568781375885, -1.9676878452301025, -0.276399701833725, -1.612253189086914, 1.00288987159729, -0.7603033781051636, -0.08130095899105072, 1.6039917469024658, -0.7620193958282471, 0.759360671043396, 0.10418631881475449, 0.5307202339172363, 2.0691871643066406, -2.352811574935913, -1.6262145042419434, -0.474073588848114, 1.1704527139663696, -1.4309489727020264, -2.27817702293396, 0.7297410368919373, 0.16511647403240204, -0.23822341859340668, 0.685783863067627, -0.8327467441558838, -0.392826646566391, 0.6070143580436707, 0.5177775025367737, -0.20783883333206177, 1.1301568746566772, 1.3293724060058594, 1.2250877618789673, -1.6178979873657227, -1.2262980937957764, 0.026659665629267693, -2.1713104248046875, 0.7323548793792725, 0.6672484278678894, 0.6036314368247986, 0.8190091252326965, -0.2512771189212799, -1.880602240562439, 0.13772521913051605, -0.13254252076148987, 0.35890379548072815, 1.0807875394821167, 2.375208854675293, -0.9189822673797607, 1.2065191268920898, 0.5342784523963928, -1.5757519006729126, 0.09851465374231339, 0.3029076159000397, 0.38041242957115173, -0.5346462726593018, 0.9276013970375061, -0.21714971959590912, -1.9598071575164795, 0.2500750422477722, 0.8031461834907532, 0.7837749719619751, 0.8447240591049194, 0.8667970895767212, -0.16033761203289032, -0.29822131991386414, 0.16424106061458588, 0.19231662154197693, -0.5032452940940857, 0.1347983032464981, 1.5500938892364502, -1.1364200115203857, -0.9340903162956238, -0.020710736513137817, 0.20261283218860626, -2.1525042057037354, -1.0938067436218262, -1.0149577856063843, -0.5457888245582581, -2.238477945327759, -0.5336954593658447, -1.8661648035049438, 0.1773684024810791, -2.085038661956787, 0.42166200280189514, 0.1214689388871193, 1.8375486135482788, -0.507271945476532, -0.9809824228286743, 1.2821522951126099, -0.41394558548927307, 0.4088127911090851, 0.7285826802253723, -0.14481714367866516, -1.8763082027435303, 0.19650082290172577, 0.5974599123001099, -1.8616986274719238, -0.44473907351493835, -0.7675560712814331, 0.46202290058135986, 1.2547240257263184, -1.123530626296997, -0.9099634289741516, -1.090590238571167], [-0.3162306249141693, -0.153779998421669, -0.6334304809570312, -0.7543436288833618, -0.4048815965652466, -1.180226445198059, -0.5195719599723816, -0.12558656930923462, 1.1184167861938477, -0.8712181448936462, -0.005975756794214249, 0.2739439904689789, 2.5689151287078857, -2.022331953048706, -1.160978078842163, 2.813981056213379, 0.8872988224029541, -0.0796402245759964, 0.465017169713974, -0.13754023611545563, -0.6771173477172852, -0.7322629690170288, -0.11137151718139648, 0.9915911555290222, -0.5069049000740051, -0.6790973544120789, -1.1179059743881226, 0.33148059248924255, 0.2804984450340271, -0.8653160333633423, 0.7341504693031311, 1.0645802021026611, -1.171674132347107, 1.5326935052871704, 0.10638201981782913, -1.6604806184768677, 1.2703721523284912, 0.8793841600418091, -0.052906300872564316, -0.5885067582130432, 1.9595874547958374, -0.025638561695814133, -0.6163375377655029, -1.1274738311767578, 2.1538491249084473, 1.0616849660873413, 0.3536049723625183, -0.376246839761734, 0.08328232169151306, 0.8144335746765137, 0.014226991683244705, -0.2836810052394867, -0.8195034861564636, 0.17413592338562012, 0.12717172503471375, 0.6257282495498657, -0.8956537842750549, 0.37559106945991516, 1.1384103298187256, -0.9622553586959839, 0.6782591342926025, -0.6911979913711548, 0.07365411520004272, 1.307323694229126, 0.1499178260564804, -0.2094903588294983, -1.3337208032608032, -0.912819504737854, 0.3111427128314972, 0.6047066450119019, -0.9328791499137878, 1.210476040840149, 1.2114627361297607, -1.2636362314224243, -0.6609982252120972, 0.3935670256614685, 0.36544302105903625, 0.2592214345932007, 0.04050139710307121, 0.2743974030017853, -2.265773296356201, -0.864595353603363, -0.1328916698694229, 0.2802596092224121, 1.1408061981201172, -0.6186432838439941, 1.1144304275512695, -1.2292299270629883, -0.9453728199005127, -0.21207237243652344, -0.4707627296447754, -1.4146075248718262, -0.2918684184551239, -0.3386579155921936, -0.668483316898346, -0.7632165551185608, -0.4369839131832123, -0.2773902416229248, -0.043466147035360336, 2.299128770828247, -1.089271068572998, 0.6918079853057861, -0.47460582852363586, 0.9776877760887146, 0.49153655767440796, 1.1495614051818848, 0.42064639925956726, -1.060232162475586, -0.6271965503692627, 0.49753713607788086, 0.8545935750007629, -2.184860944747925, 1.3475441932678223, -0.12797407805919647, -0.6999762654304504, -1.9704753160476685, 0.41210484504699707, -0.09231330454349518, -1.4314512014389038, 0.7615250945091248, -0.9078361988067627, -0.08025650680065155, -1.1464570760726929, 2.015545606613159, -1.0776625871658325, -0.4522804915904999, 1.7407945394515991, 1.1346896886825562], [0.30584579706192017, -0.007461026310920715, -0.15187591314315796, -0.9203122854232788, 0.8714451789855957, -0.26580968499183655, 0.4331902265548706, 1.0272241830825806, 0.8711145520210266, -1.217873454093933, -0.6045340299606323, -0.8466005325317383, 0.8323954343795776, 0.18038463592529297, -0.6310415267944336, 0.765936553478241, 2.694854259490967, 1.7331289052963257, -1.1504508256912231, 1.6655296087265015, -0.708206832408905, -0.23735769093036652, 0.48879700899124146, -1.686225175857544, 1.953282356262207, -0.7530363202095032, 2.202134609222412, 1.2956572771072388, -1.0931119918823242, 0.536541223526001, -0.8579248785972595, 0.7208529710769653, -1.0500729084014893, 0.37169548869132996, -0.8736138939857483, -0.8225072026252747, 0.38437607884407043, 0.4374261498451233, 1.4364160299301147, 1.0008907318115234, -0.45305293798446655, -0.11148551106452942, -0.03147326037287712, -0.5620281100273132, 0.46987423300743103, -0.338703989982605, 1.9937896728515625, 0.5770948529243469, 0.24293579161167145, -0.15325886011123657, 0.15152998268604279, 1.5844680070877075, 0.5014476776123047, 0.8930550217628479, -0.10503075271844864, 0.6793599128723145, 0.3608526885509491, 0.033380020409822464, -1.9837439060211182, -0.08006632328033447, 0.21340319514274597, -0.4521769881248474, -0.3762548565864563, -1.0310297012329102, 0.6269415020942688, -0.9046062231063843, -0.6855618357658386, -2.0321431159973145, 2.294665575027466, -0.5553957223892212, 1.0090137720108032, -0.27468135952949524, -1.5257123708724976, 0.4571525454521179, 0.021093010902404785, 0.38237056136131287, 1.2831381559371948, -1.0095797777175903, 0.039683423936367035, -0.12779909372329712, 1.8338342905044556, -0.09990805387496948, -0.7889894843101501, -1.0369271039962769, 0.4779787063598633, -1.659118890762329, -0.05115335434675217, -1.3596270084381104, 1.1274209022521973, -0.9098422527313232, -0.6922556161880493, 0.7134091258049011, -0.5137969255447388, 0.43538784980773926, -1.6836339235305786, 1.2923402786254883, 0.5723896026611328, -0.8455038070678711, 0.9484691023826599, 0.3517141044139862, 0.11680921167135239, 1.059810996055603, 1.3949559926986694, 0.46666815876960754, 0.7666693925857544, -0.1401194930076599, -1.659545660018921, -0.6108800768852234, 0.3854672908782959, 1.181225299835205, -1.8225820064544678, -1.5757067203521729, -0.12668032944202423, -1.722712516784668, 0.6945755481719971, 0.24767623841762543, 1.3177595138549805, -0.3462464213371277, -1.2548654079437256, -1.3464571237564087, 1.374668002128601, -2.6792943477630615, 1.2267463207244873, 0.9651469588279724, 0.9390618801116943, -1.4349029064178467, 1.6203389167785645, -0.5792756080627441], [-2.3051929473876953, -1.4560964107513428, -1.0779153108596802, 1.164103388786316, 0.02138538472354412, -0.11996650695800781, -1.4811015129089355, 0.838236927986145, -0.9641114473342896, -0.20235519111156464, -0.6225857138633728, 0.19848504662513733, 0.9965627789497375, 0.7097697257995605, 1.2002344131469727, -0.2750072479248047, -0.4197622239589691, 1.3132116794586182, -0.2191816121339798, -0.5920640230178833, -0.056938380002975464, 0.1177317425608635, -0.8100337982177734, 0.06853239983320236, -0.6694826483726501, 0.4178226888179779, -0.2883361577987671, 0.30620095133781433, -2.9418785572052, -0.21999506652355194, 0.2678956091403961, 0.09022965282201767, -0.6262716054916382, 1.446523666381836, -0.6579839587211609, -0.05543730407953262, 0.535577118396759, -1.6557503938674927, -1.1376729011535645, -2.4067721366882324, 0.6179801821708679, -0.5668403506278992, 0.10256725549697876, 0.4025351405143738, 1.0775103569030762, 0.6373171806335449, 0.5484556555747986, -0.21673329174518585, -2.748528242111206, -1.1657142639160156, 0.5388422012329102, 0.4698283076286316, -1.0898761749267578, -0.29789310693740845, 0.7363632321357727, -0.08722571283578873, 0.26778092980384827, -0.9804615378379822, 0.10660941153764725, -1.0125620365142822, -0.8239621520042419, -0.572031557559967, -1.8432955741882324, 0.3928402364253998, -0.24854564666748047, -0.9520772099494934, 0.49376925826072693, 0.4925670623779297, -0.784247100353241, -0.48933374881744385, 0.97205650806427, 0.166947603225708, 0.20912335813045502, 0.595085620880127, 0.6119389533996582, -0.6142869591712952, 0.45371997356414795, -0.7399221658706665, 1.0864964723587036, 0.053680699318647385, -1.0923261642456055, 0.16237953305244446, 1.1297240257263184, 0.7257342338562012, -1.5039173364639282, 1.2105926275253296, 0.4516913890838623, -2.388582229614258, 1.0182701349258423, 1.401523470878601, 0.6959302425384521, -0.358316034078598, 1.5927573442459106, 1.0935629606246948, 0.3783894181251526, -1.2105025053024292, 1.0965715646743774, -0.6170107126235962, -0.2915710210800171, -0.753706693649292, 0.5233649611473083, 0.43334880471229553, -0.2918502688407898, 1.9451040029525757, 1.4762682914733887, 1.5027996301651, 1.003888726234436, 1.3792093992233276, -0.38435617089271545, -0.5448910593986511, -0.6322416067123413, -0.9641347527503967, 0.3837394118309021, -0.8618155717849731, 0.863814651966095, -0.6948574781417847, -1.3303884267807007, -0.2057233303785324, 1.6647721529006958, -1.9279091358184814, 0.011567054316401482, 0.8335432410240173, 1.5008957386016846, -0.28824248909950256, -0.509072482585907, 1.4617738723754883, -1.1305574178695679, 1.5530204772949219], [0.36317208409309387, 0.32020747661590576, -1.1683802604675293, 2.20743989944458, -0.5186344981193542, 1.0462360382080078, -0.06604904681444168, -1.404363989830017, -0.6938921213150024, -0.5711743235588074, 0.5076311826705933, 0.7572576403617859, 0.6534256339073181, 1.5105068683624268, -0.5979573726654053, -0.6698784828186035, 1.297296166419983, -2.4145195484161377, -0.13424062728881836, -0.9456708431243896, 1.7639466524124146, -0.11871735751628876, -0.47528284788131714, 0.1511947065591812, -0.6995465755462646, -0.3964887261390686, -0.04588424414396286, -1.88727605342865, -0.9980972409248352, -0.25748905539512634, -1.0606743097305298, -0.6736065149307251, -1.2747465372085571, 0.8539236783981323, 0.7242814302444458, 1.4824904203414917, -0.5041468143463135, -1.9395002126693726, -0.2053806483745575, 0.3232354521751404, -0.04166625812649727, -0.47823089361190796, 2.8321099281311035, -0.5972963571548462, -0.9535308480262756, 0.706425666809082, 0.08249948173761368, -1.5848792791366577, 0.3091321885585785, -0.26573798060417175, 0.09072030335664749, -0.6242237687110901, 0.5786918997764587, -0.8807440996170044, 1.550052523612976, 1.04304039478302, -0.38993123173713684, -0.09044413268566132, 2.5516769886016846, 0.261260449886322, -0.41347452998161316, -0.15643921494483948, 1.2404152154922485, 0.23998293280601501, 0.6802027225494385, 1.4594807624816895, 0.05537432059645653, -1.0292792320251465, 1.244712233543396, -0.9909210801124573, -0.8128551244735718, -1.382880687713623, 0.8349146246910095, 2.1527156829833984, -1.4940320253372192, -2.698326349258423, 0.010051093995571136, -0.5209125876426697, -1.0431702136993408, 1.2507665157318115, -0.5009889602661133, -1.2510724067687988, 1.3619449138641357, 0.1844508796930313, 1.2150510549545288, -0.5246812701225281, 0.1824837028980255, 0.8446640968322754, 0.4001229703426361, -0.46287938952445984, 0.7247135639190674, -1.8567074537277222, -0.6780344247817993, 0.9348533749580383, 0.6976017355918884, -0.7526137828826904, 0.11944538354873657, 0.14362820982933044, 1.0155056715011597, 0.2874428927898407, 0.5630438923835754, -0.9254271388053894, -0.45338699221611023, -0.6093197464942932, 0.6930806040763855, -2.321403741836548, -0.35104674100875854, -0.1211935430765152, 1.294774055480957, -0.2896070182323456, 0.8154228329658508, -0.28891271352767944, -0.12541741132736206, -0.4382709562778473, 0.3753426969051361, -0.340006023645401, 1.5105887651443481, -1.4036602973937988, -0.7006016373634338, 0.0566876195371151, 0.5287162065505981, -1.3883960247039795, -0.19333003461360931, 0.5271168947219849, -0.25882840156555176, -0.9600169658660889, 0.9346587657928467, 1.732216715812683], [-0.44671162962913513, 0.5963215827941895, 0.5446339845657349, 0.28537359833717346, 0.8884989619255066, 1.2865490913391113, -0.37590205669403076, -2.0827622413635254, 0.3318168520927429, -1.499236822128296, 1.3618539571762085, 0.038360659033060074, 1.9213011264801025, 0.2637971043586731, 2.06669545173645, -0.19278591871261597, 1.216695785522461, 0.5308853983879089, 0.9556363821029663, 1.3839104175567627, -0.03892279416322708, -0.15500923991203308, 0.1286378651857376, -0.5208866596221924, -0.5846142768859863, 0.0005931448540650308, -0.2790609896183014, -1.4009653329849243, -1.6038607358932495, -0.9000683426856995, -0.753866970539093, -0.8658168315887451, -0.9276467561721802, 1.908970832824707, -0.8714859485626221, -0.6790194511413574, 0.8271569609642029, -1.0053555965423584, -0.7674031257629395, -2.294611930847168, 0.7530611157417297, -1.118327021598816, 0.041263386607170105, 0.0625811368227005, -1.595603585243225, 1.3754081726074219, -0.26552248001098633, 0.46552687883377075, -0.1494404375553131, 1.7614023685455322, -0.39971446990966797, 0.45219460129737854, -0.2784327566623688, 0.42216575145721436, 0.07277072221040726, 0.5126175880432129, -0.7615494728088379, 0.2944451570510864, 0.7303008437156677, -0.2604246139526367, 1.6688599586486816, -0.36036941409111023, 0.681160569190979, 0.20843608677387238, -1.939094066619873, 0.8139531016349792, -1.3341573476791382, 0.20997263491153717, 0.0479913204908371, -1.0487380027770996, 0.24210014939308167, 1.8060863018035889, 1.3287111520767212, 1.187049150466919, -0.9805488586425781, -0.16382655501365662, 0.4338676333427429, 0.6779592037200928, -0.25233855843544006, 1.7819362878799438, -0.38582512736320496, -0.47218313813209534, 1.3940062522888184, -1.371500849723816, 0.9633839130401611, 1.4876320362091064, -0.14487679302692413, 1.0913150310516357, -1.0569037199020386, 0.17184217274188995, 0.17398414015769958, 0.550950825214386, 1.2954219579696655, -0.7819579243659973, 1.2854113578796387, -0.23778842389583588, -0.3227483034133911, 1.075510025024414, 0.5189072489738464, -0.38626644015312195, -0.08117461949586868, -0.8375645875930786, 1.002989649772644, -0.024875469505786896, 0.6138055920600891, -0.1798483431339264, 0.5375568866729736, -1.7512054443359375, -0.09860214591026306, 1.6310144662857056, 0.9247118234634399, -1.6667768955230713, 0.06895927339792252, 0.49516263604164124, -1.2458852529525757, -1.19141685962677, 0.8449040651321411, 0.5892665386199951, 1.1435014009475708, -1.416942834854126, 0.3367483615875244, -0.9375932216644287, 0.8486459851264954, -0.04537453502416611, -1.0355641841888428, 0.8733097314834595, 1.035853624343872, 1.3495304584503174], [0.8279181122779846, 1.4717466831207275, -0.11481758952140808, -1.091407060623169, 0.6667123436927795, -1.120306134223938, 1.7596728801727295, -0.006034274119883776, 0.0993160679936409, -0.8226660490036011, -1.3057811260223389, -2.738558769226074, -1.1397594213485718, -1.2977309226989746, -0.883438766002655, -0.8295480608940125, -0.14313770830631256, 0.08303488790988922, -0.145940363407135, 0.4991164207458496, 0.8356668949127197, -0.2532272934913635, 1.348040223121643, 0.7272783517837524, -0.08457692712545395, 0.22426089644432068, -0.7179807424545288, 0.3257569670677185, 0.5057400465011597, -1.7359100580215454, -0.9536179304122925, 0.9765622019767761, 0.07403309643268585, 2.197739839553833, 0.8121309876441956, -0.02556357905268669, 0.2734127938747406, -1.0975937843322754, -1.5660290718078613, -1.0681283473968506, -0.6494335532188416, -1.3499118089675903, 0.6114287972450256, -0.6598582863807678, 0.6528928279876709, -0.46340179443359375, -0.4157063663005829, 0.39045843482017517, 0.2093794345855713, -0.01712833344936371, -1.8673392534255981, -0.9411104917526245, 0.5093258023262024, 0.47655314207077026, 0.6542969346046448, 0.02781350165605545, 1.870424509048462, -0.4550381302833557, 0.5945087671279907, 1.6272493600845337, -0.2563038766384125, -1.4076608419418335, 0.2959774136543274, -0.33993735909461975, -0.38104739785194397, -0.5524567365646362, 0.32924672961235046, 2.105499744415283, 1.0599044561386108, 0.8013883233070374, -0.17298981547355652, -0.5666952729225159, -0.9111233353614807, -0.4712781608104706, -1.3172314167022705, -1.718995451927185, 0.657513439655304, -0.13211368024349213, 2.237535238265991, -0.47363972663879395, -1.2713123559951782, 0.09564174711704254, -1.059044361114502, 0.5072075128555298, 0.05826527625322342, -1.3822729587554932, -0.7164633870124817, -0.03503299131989479, 1.4259506464004517, 0.41129565238952637, -0.9437894225120544, 0.4341103136539459, -0.3092705011367798, 1.4926841259002686, -0.18965131044387817, 0.3729722797870636, 0.9984327554702759, 0.4083991050720215, 0.725372850894928, -0.5968465805053711, 0.040041301399469376, 0.41445818543434143, 0.1766815185546875, -0.14635618031024933, -1.5098321437835693, 1.1618759632110596, -0.10851617902517319, -0.9953261613845825, -0.7230258584022522, 0.9678122997283936, -0.325173944234848, 1.4153919219970703, -0.8056815266609192, -0.632988691329956, -0.7276731729507446, -1.22086501121521, 1.1856775283813477, -0.48652851581573486, -2.0147368907928467, 0.9272302389144897, 0.486803263425827, -0.35940074920654297, 0.37560948729515076, -1.9431297779083252, 0.22418396174907684, -0.3405890464782715, -0.20668788254261017, -1.0842300653457642], [-1.13017737865448, -0.3226507604122162, -1.8915313482284546, 1.4417181015014648, -0.5926794409751892, 2.5180439949035645, -1.7564616203308105, -0.5761958956718445, -0.47886037826538086, 0.23314736783504486, -0.3433701694011688, 0.7865749001502991, -0.4421885013580322, -1.0129895210266113, -0.9105768799781799, 0.41600218415260315, -1.86505925655365, -0.37264400720596313, -0.10206270217895508, -1.0086275339126587, -0.15385107696056366, 0.21572306752204895, -0.252156138420105, -0.421860009431839, 0.33715203404426575, -0.10825543105602264, 1.0022404193878174, -0.003612851956859231, -1.0826703310012817, -0.2281009405851364, -0.03448288515210152, 0.6627814769744873, -0.13099631667137146, 0.06588573008775711, 0.5840811729431152, 1.492480754852295, -0.8902900815010071, 0.18108171224594116, -1.3524202108383179, 0.4233979880809784, -0.08851827681064606, 0.26123008131980896, 1.1085950136184692, 0.8376334309577942, 1.5154749155044556, -0.48500269651412964, -0.8237322568893433, -0.15969602763652802, 0.13283811509609222, -1.7463514804840088, 1.132947325706482, 0.18671339750289917, 0.80307537317276, -1.472312569618225, 0.10363627225160599, 0.6344597339630127, -1.5330039262771606, -0.44685256481170654, 0.37091192603111267, -1.3279366493225098, -1.640033483505249, -0.3852396309375763, 2.5661306381225586, -1.2603013515472412, 0.3976423740386963, 0.5379703640937805, -0.4262078106403351, -0.8133429288864136, 0.3902103006839752, 1.3545289039611816, -0.1955554038286209, -0.972843587398529, -0.5633453726768494, 1.0056699514389038, 0.7368043065071106, -0.43045276403427124, 1.0807578563690186, 0.9119852185249329, -1.0602986812591553, 0.4753350019454956, 1.4416234493255615, -0.9894146919250488, -1.0966899394989014, 1.1691919565200806, 0.9016414880752563, 0.06884687393903732, 0.7347795367240906, -0.6251561045646667, 0.09963425993919373, 0.9398435354232788, -0.890680730342865, 0.40825003385543823, -0.7250747680664062, -0.5000740885734558, -0.34788399934768677, 1.3382327556610107, 0.7459318041801453, 0.30585286021232605, -0.7587078213691711, -0.7774474620819092, 0.1416010856628418, -0.19063594937324524, -0.5015101432800293, 0.9082951545715332, 1.3196324110031128, 0.9061993956565857, 0.6970080137252808, 0.5650734305381775, -1.7927321195602417, -0.5840137004852295, -0.25690755248069763, -1.9461393356323242, 1.1289573907852173, -0.08513103425502777, 0.8113396763801575, -0.5634381771087646, 1.4151684045791626, 0.7298493981361389, 0.3472779393196106, -1.975158452987671, -0.5379190444946289, 0.9897561073303223, -0.8789926767349243, 1.1841466426849365, -1.6378426551818848, -1.4461618661880493, -1.5247410535812378, 0.5161603093147278], [-0.7497372031211853, 0.020768899470567703, -0.4982452690601349, -0.24788786470890045, -0.9706589579582214, 1.1534055471420288, 0.27326497435569763, 0.9919995069503784, -0.22339054942131042, -0.11103427410125732, 2.2846899032592773, -0.8536315560340881, -0.20855095982551575, 0.5026041865348816, 1.5019962787628174, -0.6461896896362305, 0.7410633563995361, 1.2800242900848389, -1.0232696533203125, 0.2974783182144165, 0.5046631693840027, -1.388840913772583, -0.9881691932678223, 1.8843415975570679, -0.5967313647270203, 0.5561569333076477, 0.13025711476802826, 0.07096308469772339, 0.3572627305984497, 0.3637804388999939, 1.772032380104065, -0.863693118095398, 0.11608601361513138, 0.010227564722299576, -0.2884978652000427, -0.76735919713974, -0.6073235869407654, 0.11795850843191147, -0.3663865327835083, -0.2580265700817108, 0.9440988302230835, 0.6587026715278625, 0.6100223064422607, -0.21809761226177216, 0.27190929651260376, -0.6365501284599304, 0.5715307593345642, 2.1492245197296143, -0.41628000140190125, -0.5230109095573425, 0.4138314723968506, 0.4006400406360626, 0.8526360988616943, 0.4240185022354126, -0.1576518714427948, 1.2352646589279175, 0.7832242250442505, -0.1820044219493866, -0.5087327361106873, 1.6149423122406006, 1.3071861267089844, -1.8736660480499268, 1.5767881870269775, -1.6013425588607788, -0.014412076212465763, -1.1536989212036133, -1.028056263923645, -1.165416955947876, 0.4136483073234558, -0.1905938684940338, -0.37425118684768677, 0.04029453918337822, 0.729019284248352, 0.33117571473121643, 0.11535250395536423, 0.9100767374038696, 0.5363033413887024, -0.09066974371671677, 2.3484206199645996, 0.9702959656715393, 1.7363606691360474, -1.3859517574310303, -0.3961060047149658, -0.5435184240341187, 0.16830316185951233, -0.06074826419353485, 1.4575608968734741, 0.7958841919898987, 0.882573664188385, 0.1696040779352188, 1.2177809476852417, -0.49301448464393616, -0.3865666091442108, 0.07170218229293823, -0.3527171015739441, 1.6744229793548584, 0.019288504496216774, -0.48539823293685913, 0.12883147597312927, 0.879690945148468, 0.21031403541564941, 1.7773363590240479, -0.3401544988155365, 0.6154214143753052, -0.2898995578289032, -0.4800534248352051, 0.2711564302444458, -0.2576637268066406, -1.8362957239151, 0.5264521241188049, -0.8928830027580261, 0.8538239598274231, 0.6849604845046997, -2.1879801750183105, 0.16292302310466766, 0.33455461263656616, -0.9036839604377747, 0.529952347278595, -1.6075226068496704, -1.3073878288269043, 0.08549808710813522, -0.1363513320684433, 0.5185369253158569, 0.5870123505592346, -0.21179695427417755, -0.22276125848293304, -0.11768224090337753, -1.8993422985076904], [1.6611613035202026, 0.27454760670661926, -0.8616747856140137, -0.4398934841156006, -0.38228607177734375, 0.7277445197105408, -1.0824323892593384, -1.4573757648468018, -0.3065330386161804, -1.8016127347946167, 0.22524648904800415, -1.4670753479003906, -0.08925991505384445, -0.3392091691493988, 0.6687686443328857, -1.472476601600647, 0.5275208353996277, -0.6114144921302795, -0.13745956122875214, -1.4536939859390259, -0.06740012764930725, 0.7134738564491272, 0.9949864149093628, 0.4389553666114807, -0.6982109546661377, 0.6565921306610107, 0.4685560464859009, 0.5319212675094604, 0.22899116575717926, -0.23016244173049927, 0.9486114978790283, 1.1316944360733032, -0.10067402571439743, -0.29323214292526245, -0.2873443067073822, 0.5056150555610657, -0.21031345427036285, 0.760529637336731, 0.26255083084106445, -0.4943870007991791, 0.7861935496330261, 1.109734058380127, -0.6515089869499207, 0.11239025741815567, -1.2238012552261353, 0.46990612149238586, -1.021134614944458, 0.03451596200466156, 0.9674006104469299, 1.2114886045455933, -0.7032313942909241, 0.33614248037338257, -0.5075384974479675, 0.2887318730354309, -0.3438924551010132, -0.0030019518453627825, 1.7061597108840942, -1.1182345151901245, -0.19862164556980133, 0.6656420826911926, -0.27215564250946045, -2.7495806217193604, -2.014594078063965, 0.07342948019504547, 0.24755500257015228, -0.24425944685935974, 1.967306137084961, -0.7550663352012634, 0.16925403475761414, 0.5453162789344788, 1.2399277687072754, 0.6971776485443115, -1.4918859004974365, 0.11561701446771622, 0.953750729560852, 0.5767499208450317, 1.0338197946548462, 0.009660274721682072, 1.144326090812683, 0.7173191905021667, -0.4401029348373413, 2.289584159851074, 1.0802773237228394, 1.8995261192321777, 1.8456995487213135, -0.11346252262592316, 0.5246151089668274, 0.35665443539619446, 1.7094124555587769, -0.48315978050231934, 1.4142159223556519, -1.2398335933685303, 0.5389122366905212, -0.3747803568840027, -0.028225749731063843, 0.0544021837413311, -0.9354806542396545, -0.8184728026390076, -0.23704098165035248, -0.10343378037214279, -0.4072885513305664, 0.9801688194274902, -0.16954012215137482, -1.1141917705535889, 0.40266847610473633, -2.289980411529541, 1.2771313190460205, 0.2060726135969162, 1.6681785583496094, 0.7994407415390015, 0.28730472922325134, -1.0335084199905396, -0.3476661145687103, -0.45050135254859924, 1.8468453884124756, -0.7803038358688354, -1.0130270719528198, 0.41599249839782715, 0.21002346277236938, 1.0721733570098877, 0.05545240640640259, 2.202354669570923, -0.42565852403640747, 1.8249672651290894, -0.6492951512336731, -0.19871404767036438, 0.9688535928726196, -0.5986543297767639], [1.4066702127456665, -1.5600700378417969, -0.8530480265617371, 0.8934707641601562, 1.3679840564727783, 0.4297671318054199, -0.4279631972312927, 1.2057706117630005, -0.31716468930244446, 0.5558509230613708, -0.9567686319351196, -0.1271539181470871, -1.117632269859314, -0.02881649136543274, 0.19280970096588135, 1.8005684614181519, 0.25341796875, 0.2379802018404007, -1.5868544578552246, 0.037120070308446884, -1.374837875366211, -1.0588535070419312, -0.209535613656044, -0.16604973375797272, 1.9741920232772827, 1.2311865091323853, 0.4205467700958252, 0.0978347510099411, 1.0983034372329712, 0.3328370451927185, -0.188731387257576, -2.117600202560425, 0.18009208142757416, -0.7254273891448975, -0.4755299687385559, -0.5939142107963562, -0.7435917854309082, -0.18805363774299622, 0.08173826336860657, -0.7020349502563477, -0.4208150804042816, -0.1510254144668579, 0.655769407749176, -0.40388572216033936, 0.7305698990821838, 0.8814557790756226, -1.05672287940979, 0.21309611201286316, 0.7571805119514465, 2.086130380630493, 2.0487935543060303, 0.1501472443342209, -0.026096342131495476, 0.08604362607002258, 1.3194372653961182, -1.7140047550201416, -1.2365797758102417, 0.029158344492316246, 1.0907011032104492, 0.17727743089199066, 0.6541743874549866, -1.1013059616088867, 0.5515854954719543, -1.213113784790039, -0.15165671706199646, 1.293415904045105, 1.2702962160110474, 1.1798746585845947, 0.9776555895805359, 1.1978694200515747, -2.0730972290039062, 0.516056478023529, -0.48477551341056824, 0.8143666982650757, 0.3456916809082031, 0.3690970540046692, 2.4101898670196533, 0.20371203124523163, 0.2260202318429947, 1.5201798677444458, -0.4803781807422638, 1.1553518772125244, -0.766377866268158, -0.478497713804245, 0.18973380327224731, -0.7640892863273621, -1.4604978561401367, -0.531696081161499, -1.3840733766555786, 0.03487730771303177, 0.12875241041183472, -1.093432903289795, -1.353597640991211, -0.21879985928535461, -0.2594490945339203, -1.0631322860717773, -0.9809680581092834, -1.0519479513168335, 1.1139171123504639, -0.1628425419330597, 0.2559429705142975, 1.1615186929702759, 2.0807645320892334, -0.6886627674102783, -0.43737858533859253, 0.25965094566345215, -0.9549003839492798, -0.4089813232421875, -0.08874606341123581, 1.301500916481018, 1.420589804649353, 0.23950566351413727, -0.9564938545227051, -0.20397892594337463, 0.9428942203521729, -0.06387756019830704, -2.032460927963257, -0.9672929048538208, 2.0414695739746094, 0.9649145007133484, -0.6509749889373779, 0.5665726065635681, -1.0556410551071167, -0.597812294960022, 0.1416475474834442, 0.2367793768644333, -0.3082880973815918, 1.2397528886795044], [-1.1726137399673462, 0.03616990149021149, 0.2276403158903122, 0.6857153177261353, 0.16639059782028198, 0.7841417193412781, 0.622087299823761, -0.34399479627609253, 0.14748519659042358, -0.9866859316825867, 1.068617582321167, 0.33349478244781494, 0.9837000966072083, -0.3632056415081024, -0.2506186068058014, 0.6153289675712585, -0.4825126528739929, 1.6567256450653076, 0.562950611114502, -0.9125733375549316, -0.43957868218421936, -0.8026522994041443, 0.7484294772148132, -0.6912389993667603, -0.45090949535369873, 1.329390525817871, 0.8263789415359497, -0.6463264226913452, 2.9631521701812744, 0.7040735483169556, 0.5820746421813965, 0.7787388563156128, 0.20264656841754913, -0.41079023480415344, -1.3089728355407715, -0.8892883658409119, -0.5080087184906006, -0.7354444265365601, -0.16190044581890106, -0.16211684048175812, 2.3753750324249268, 1.9977747201919556, -1.5796276330947876, 0.3178984820842743, -2.065349817276001, 0.6920856237411499, -0.6202725172042847, -0.5122167468070984, -1.956885814666748, -0.5753592848777771, 1.3696480989456177, 1.1793361902236938, 0.8247647881507874, 0.6537902355194092, -0.29566702246665955, -1.122719168663025, -0.11258514970541, 2.0664432048797607, 0.06016841158270836, 0.48124733567237854, -0.5242534279823303, -0.43923506140708923, -0.3505937159061432, -0.8904129862785339, 1.1753655672073364, 1.6634615659713745, -0.1994473934173584, 0.7187180519104004, 1.4334633350372314, -0.3490099310874939, -0.189827561378479, 0.023974668234586716, -0.4568924903869629, -1.2661011219024658, 0.16300275921821594, -0.18535470962524414, 0.0557464063167572, -0.30594006180763245, 0.48642417788505554, -0.7790377140045166, 0.642889678478241, -0.3769674599170685, -1.732813835144043, 1.4001288414001465, -0.27758345007896423, -1.008286952972412, -0.6426932215690613, 0.09921237081289291, -2.2806296348571777, 1.018033504486084, 0.78961181640625, -1.1760140657424927, -0.18003927171230316, 1.2340261936187744, -0.16901400685310364, -2.0340957641601562, -1.1095589399337769, -0.08291592448949814, -0.22457031905651093, 0.2790677547454834, 0.4120752215385437, 1.9377586841583252, 0.350851446390152, 0.7145500183105469, -0.8668699860572815, -1.2412021160125732, 0.20471040904521942, -0.9075828194618225, 0.830832302570343, 0.39296582341194153, 1.540667176246643, -0.34465253353118896, 1.6747864484786987, 1.3902955055236816, 1.2881875038146973, -0.14884313941001892, -2.683979034423828, 0.7977630496025085, -0.1697336882352829, -0.6865692138671875, 2.395920515060425, 0.6412568092346191, -0.8367229700088501, 0.04410610720515251, -1.633581519126892, 0.026146724820137024, -0.012538906186819077, -1.0551666021347046], [1.705573558807373, 0.3328877389431, 0.6837785243988037, -1.3270920515060425, -1.4720107316970825, -0.5139620900154114, 0.6533512473106384, -0.24016012251377106, -1.1981487274169922, -1.4966667890548706, 0.7322507500648499, 0.1343095898628235, 0.2530372738838196, 1.1439818143844604, 2.9419238567352295, 0.44387826323509216, -0.22997267544269562, -0.6905052661895752, 0.054335977882146835, 1.3155497312545776, -0.9697125554084778, -0.4870404899120331, -0.6459605097770691, -0.3824754059314728, 0.6432668566703796, -1.2911503314971924, 0.11950182169675827, 0.2718378007411957, -0.4907180070877075, -0.32596898078918457, 0.39518237113952637, 1.0297194719314575, 1.4498865604400635, 0.0419122688472271, 0.46977517008781433, -0.359698623418808, 0.13180918991565704, 0.009092897176742554, -0.024432417005300522, 0.4323274791240692, -0.7226628065109253, 1.2016396522521973, 0.41281941533088684, 1.1670314073562622, -1.1507642269134521, -1.431128740310669, 0.43808457255363464, -0.4435292184352875, 0.2629442512989044, 0.9650930166244507, -0.20774434506893158, -0.24810083210468292, 0.7756487131118774, 0.4467051029205322, -1.3400118350982666, -0.4908388555049896, -0.49523118138313293, 0.9601767659187317, 0.6103543043136597, 0.11189482361078262, -1.5045444965362549, -0.8335727453231812, 0.1062355637550354, -1.0002566576004028, 1.114168643951416, -2.196223020553589, -1.4233802556991577, 1.9087682962417603, -0.11139164865016937, 0.319439560174942, -0.5234360694885254, -1.001319408416748, -0.886688768863678, -0.5792179703712463, -1.1126147508621216, 0.4128602147102356, 0.5076548457145691, 0.07802903652191162, 1.3817119598388672, -0.37988296151161194, -0.4318879246711731, -0.0212019644677639, 2.132474422454834, 0.8998541235923767, -1.5013842582702637, 2.2608323097229004, -0.14303059875965118, 2.1666629314422607, 0.38752931356430054, 0.9641902446746826, 0.9037487506866455, 0.6023859977722168, -0.17174139618873596, -0.8225910067558289, -1.113370656967163, 0.16875840723514557, 1.365899682044983, -1.421290636062622, -0.6054785251617432, 0.3100733757019043, 0.7486861944198608, -1.5233395099639893, -0.16638989746570587, -0.657257616519928, -1.2297075986862183, 1.4721640348434448, 0.8883671164512634, 1.380977988243103, -0.6762915849685669, -0.5420098900794983, -1.4225833415985107, -0.13540390133857727, 0.21292290091514587, 0.46044808626174927, -0.7656480073928833, -0.22064025700092316, -0.8764063119888306, -2.3469443321228027, -0.7130388617515564, 0.6587048172950745, 0.5714683532714844, -0.5886181592941284, -0.10457241535186768, 0.12411856651306152, -0.35856661200523376, 0.4586373269557953, 0.8935689330101013, -1.3908874988555908], [0.2779899537563324, 0.12399481236934662, -0.08072568476200104, -0.9621537923812866, 1.5581845045089722, -0.6996275782585144, 0.01617586798965931, -0.1423673778772354, -1.473017930984497, 0.5026914477348328, 0.39040666818618774, -0.16264526546001434, 0.19340571761131287, -1.4797683954238892, -0.9118058681488037, 2.205519676208496, -1.0120567083358765, -0.9207714200019836, 0.04114184528589249, 1.00640869140625, -0.3509208559989929, 0.16739702224731445, -0.35610833764076233, -0.4303056597709656, -1.11651611328125, 1.404002070426941, 2.349329948425293, 0.12943720817565918, 0.16903255879878998, 0.18878647685050964, -1.055906891822815, 1.8719133138656616, -0.8905395269393921, 1.1685868501663208, -0.3699711263179779, 1.4922304153442383, -0.5249174237251282, -0.4854266941547394, -1.1520729064941406, -1.1039583683013916, -0.14015501737594604, -0.5844482183456421, -1.0818589925765991, 0.8840346932411194, -0.686227560043335, -0.04805842787027359, 1.125749945640564, 0.5594143867492676, 1.1954509019851685, 0.265363484621048, 0.5232107043266296, -1.5790369510650635, 0.14203551411628723, -0.16837064921855927, -1.3121237754821777, 0.05805171653628349, -1.9352936744689941, 2.9202089309692383, -0.19870001077651978, -0.08926226943731308, 0.2620907127857208, 0.09099184721708298, -0.2357722967863083, 1.1564135551452637, 0.36643725633621216, 0.6313817501068115, 1.6258518695831299, 0.3177926242351532, 1.1688201427459717, 0.6333438158035278, -0.18428879976272583, -0.870491623878479, -0.6963435411453247, -0.39660245180130005, -0.10208889096975327, -0.8897721767425537, -0.09283177554607391, -0.7652561664581299, -1.5224237442016602, 0.034945983439683914, -2.015547752380371, 0.6723306179046631, -0.1507142335176468, 2.068366289138794, 1.2694872617721558, -0.38283947110176086, -0.3587891161441803, -0.4507673680782318, -0.8230783343315125, -0.1558685302734375, -0.48288899660110474, 0.009677969850599766, 1.3872755765914917, 0.7327951788902283, 0.8835290670394897, -1.1274877786636353, 1.0989024639129639, -1.7378361225128174, -1.9923537969589233, -0.43265604972839355, 0.08360373973846436, -0.34588584303855896, 0.6295680999755859, -0.63059401512146, -0.4793980121612549, -0.14960330724716187, 0.547197163105011, 0.3597118854522705, -1.0130482912063599, -1.5111571550369263, 2.4167773723602295, -0.6849002838134766, -0.266470342874527, -0.30609384179115295, -0.5011794567108154, 1.0056687593460083, -0.5556548237800598, -1.7988569736480713, 0.11716028302907944, -0.06756765395402908, 0.9327123165130615, 0.1229969710111618, -0.297782838344574, -0.2147495448589325, -0.6482835412025452, -0.4737589359283447, -1.429561734199524, 1.482162594795227], [0.7688900232315063, -1.519250750541687, -0.14927300810813904, 0.4475362300872803, -0.9282896518707275, 0.27693814039230347, 0.20985250174999237, -0.4364044666290283, -0.35004734992980957, -0.10113043338060379, 0.46031010150909424, -0.983162522315979, 0.8051475882530212, 1.7757482528686523, 1.3127907514572144, 0.045170508325099945, 1.8950310945510864, 1.514037013053894, -0.644731342792511, 1.8795478343963623, -0.752263069152832, 1.9628421068191528, 0.7334738969802856, -0.1833200603723526, 0.2981320023536682, -0.8428305387496948, -1.9864684343338013, 0.500972330570221, -1.3899134397506714, -1.1793272495269775, 0.09582889825105667, 0.6427253484725952, -0.7575328946113586, 0.014166787266731262, -1.7269326448440552, -0.22387109696865082, 1.3115853071212769, -2.0075409412384033, -0.1579234004020691, -1.0950864553451538, 0.6308737993240356, 0.20425467193126678, 0.11958315968513489, 0.9012932181358337, -0.4475289583206177, 0.014572816900908947, 0.11299755424261093, -0.5165515542030334, 1.4481817483901978, 0.9613769054412842, -1.0083733797073364, 0.5897483229637146, 0.3383072018623352, 1.8488824367523193, -1.228847622871399, -0.394247442483902, 0.36688870191574097, 2.027893304824829, -1.5645081996917725, -0.19218836724758148, -0.7670560479164124, 0.3635261356830597, 0.3435949981212616, 2.042731285095215, 0.4104204475879669, 1.1672345399856567, -1.2810568809509277, 0.2113049328327179, 1.361283779144287, 0.39935559034347534, 0.037272900342941284, 0.8714231848716736, 1.7290136814117432, 0.791027307510376, -1.2733497619628906, -1.0169565677642822, 0.668252170085907, -0.7442456483840942, 0.0009441530564799905, 0.8617039918899536, 1.0299052000045776, 2.0367848873138428, 0.5922536849975586, 1.934175729751587, 0.7645523548126221, 0.5554310083389282, -0.33979156613349915, -0.3610549569129944, -0.8706512451171875, 0.2660364806652069, -0.5196660161018372, -0.7373148202896118, -1.21595299243927, 0.2750106751918793, 0.818324089050293, -1.1049916744232178, -0.7436965703964233, 1.7966351509094238, 0.8593897223472595, 2.5372369289398193, -0.9884138107299805, 0.7447780966758728, -0.5156227946281433, -1.0195186138153076, -0.543632984161377, 0.9690225124359131, 0.5310842394828796, 1.0597420930862427, -1.3354301452636719, -0.6757020354270935, 1.375152826309204, 0.6445749402046204, 0.07626701891422272, -2.4851231575012207, -0.36947768926620483, 0.5004541873931885, 0.7772210836410522, 1.959035873413086, -0.5655434131622314, -0.5970041155815125, -2.017228603363037, -0.8832063674926758, 0.7296742796897888, 1.0724105834960938, -0.5953763723373413, -1.301338791847229, 0.4766784906387329, -0.7562096118927002], [-0.8652184009552002, -1.3025078773498535, 2.0276577472686768, -0.44164299964904785, -1.2258915901184082, 0.10678229480981827, 0.18688322603702545, -1.0277740955352783, 0.44396933913230896, -1.1337342262268066, 0.8710108995437622, 0.20629052817821503, 0.21105772256851196, -0.7525407075881958, 1.0627154111862183, -0.5215527415275574, -0.23219077289104462, 1.2712221145629883, -0.4045295715332031, -0.5787876844406128, 1.092974066734314, -0.5892412066459656, 0.3548045754432678, -0.054969679564237595, 0.3339749872684479, 0.5903589129447937, 0.6618146300315857, 0.06749264895915985, -0.23872104287147522, -0.6896218061447144, -1.0382364988327026, 0.07902543246746063, -2.348618984222412, 0.7053989171981812, -0.17981113493442535, -1.2404483556747437, -0.2915663421154022, -1.89643394947052, 0.12997578084468842, -1.1674765348434448, 0.25170835852622986, -0.6718156337738037, 0.1764189898967743, -1.1020101308822632, 1.1985052824020386, 1.595828890800476, 0.12355343252420425, -0.39333081245422363, -0.3618377149105072, -0.6628758311271667, -0.5669334530830383, -0.8082807064056396, 0.13710527122020721, -0.29669904708862305, -0.5469157099723816, 0.6985392570495605, -0.6158304810523987, 1.0252549648284912, 0.95308518409729, 0.751991868019104, -2.1967833042144775, -0.5370402932167053, 0.8387394547462463, -1.509095549583435, -0.3027481734752655, 1.210660457611084, 2.4025039672851562, 0.8925265073776245, 0.01856282167136669, -1.7245842218399048, -1.3774093389511108, -1.3513274192810059, -1.0591769218444824, 0.8906527161598206, -1.5070395469665527, 0.2844575345516205, 0.11927014589309692, -0.001925310236401856, 1.9854605197906494, 0.9903701543807983, -0.8374748826026917, 0.08897370845079422, 2.4983935356140137, -0.3789542317390442, 0.428026407957077, -0.9900375604629517, -1.6064740419387817, 0.04095190390944481, -2.371980667114258, 1.865620493888855, 1.3934619426727295, 0.010370595380663872, 0.6323062181472778, 0.670297384262085, 1.943851351737976, 0.2838115990161896, -0.2543413043022156, 2.2085461616516113, -0.6131580471992493, 0.6442369222640991, 0.34607449173927307, -0.5086621046066284, -0.044923290610313416, 0.07665816694498062, -0.517182469367981, -0.36671552062034607, -1.0306878089904785, 0.3043346405029297, -2.2368967533111572, 0.1124735027551651, 0.6718354821205139, 0.26694440841674805, -0.2645612061023712, -1.7687283754348755, 0.6091997623443604, 1.0972036123275757, 0.445222407579422, -0.38407284021377563, 1.6404238939285278, 1.0650273561477661, -0.41645240783691406, 0.4658675491809845, 0.13016371428966522, 0.8142763376235962, -0.8783025741577148, -0.33588019013404846, 1.668960452079773, -0.0010911066783592105], [0.2829523980617523, 1.409751296043396, -0.38385009765625, -2.155599594116211, -0.657363772392273, 0.07734070718288422, 0.8340303301811218, -0.7098863124847412, 0.16601026058197021, 0.725423276424408, -0.5674207806587219, -0.36014324426651, -0.6794856190681458, -1.4804174900054932, -0.10802499949932098, -0.06256361305713654, -0.4332694113254547, -0.4406983554363251, 0.15535247325897217, -0.00016978630446828902, -0.77203369140625, -1.002264380455017, -0.1560918688774109, 1.3023401498794556, -0.5401103496551514, 0.1362321674823761, 0.6659180521965027, -2.3459300994873047, 1.0976300239562988, 1.0424630641937256, 0.759845495223999, -0.5671975612640381, 0.84760582447052, 0.20165874063968658, -0.07229804247617722, 0.30292099714279175, 0.8030309081077576, 0.3933986723423004, -2.0410549640655518, 0.5674381852149963, -3.1088736057281494, 1.3713942766189575, 1.532128095626831, -0.47038429975509644, 1.5340371131896973, -0.12134502083063126, -0.34995272755622864, 0.09397423267364502, -1.365237832069397, -0.47969648241996765, -0.4033275842666626, -0.4633907079696655, -1.2703964710235596, -0.8857508301734924, -1.3523722887039185, 1.0447783470153809, -0.5738852024078369, 2.007413864135742, -0.15738177299499512, 2.1532533168792725, -0.6846428513526917, -1.192970633506775, 0.6118695735931396, 0.10985884815454483, 0.5055024027824402, -0.6274812817573547, 0.7449011206626892, 2.282595157623291, 0.5497097969055176, 0.12851229310035706, 2.567938804626465, 0.44870448112487793, -0.04481368511915207, -0.40244951844215393, -0.5115847587585449, 0.32257363200187683, 1.2359082698822021, 0.8951559662818909, -1.9258264303207397, 0.39528167247772217, -1.2080706357955933, -0.8160752654075623, 2.1968023777008057, -0.9090666770935059, -0.3450075089931488, 0.7920196056365967, -0.5677002668380737, 0.43555906414985657, -0.6687411069869995, -0.843072235584259, 1.1127370595932007, 0.6663880348205566, 2.271141290664673, 0.4293338358402252, 0.0699828490614891, -0.6952588558197021, -1.737029790878296, -1.6415833234786987, -0.4840250611305237, 1.0473510026931763, 0.8081072568893433, 0.2551625669002533, 1.3883978128433228, 0.21317198872566223, 0.4954814314842224, 1.2169976234436035, -1.0897281169891357, 0.8526005744934082, -0.13764970004558563, -1.2921894788742065, 0.0981246680021286, 0.408950537443161, -1.269175410270691, 0.15413357317447662, 1.3430010080337524, 0.5213218927383423, 0.7357739210128784, -2.029762029647827, 0.5836868286132812, -0.3266289532184601, 1.6779847145080566, 0.9968904256820679, -1.3039730787277222, -0.06360746920108795, 0.8865091800689697, -0.18481943011283875, 0.6492922306060791, -0.6655501127243042], [-1.109548807144165, 0.5037245154380798, -0.21138131618499756, 1.6554840803146362, -0.9855605959892273, -0.19332006573677063, 0.503609299659729, -1.017783761024475, 0.007105634082108736, 0.7974500060081482, 0.2865208089351654, 0.14467008411884308, -1.2116222381591797, -0.14687958359718323, -1.113371729850769, -1.2627003192901611, 0.4154277741909027, 0.42257654666900635, 1.656187891960144, -1.6969115734100342, 0.7420295476913452, -1.8340790271759033, 0.4519021213054657, 0.8163963556289673, 0.41389626264572144, -0.28464221954345703, -1.0614585876464844, -0.6290063858032227, -0.09515312314033508, -0.9620879888534546, -1.323952555656433, -1.4863916635513306, 0.6221200227737427, -0.0955924317240715, -0.13163252174854279, -0.6197255253791809, -0.22775304317474365, 0.4402996599674225, 0.2238868623971939, 0.32352226972579956, 0.40060529112815857, -0.2817515432834625, 1.486581563949585, 0.38100308179855347, -0.5989633202552795, 1.7352463006973267, 0.1684243232011795, 1.318008303642273, 0.012675235979259014, 1.1623210906982422, 2.2995896339416504, 0.5152392983436584, -0.2385733276605606, -0.6186042428016663, -0.4197527766227722, -0.798997163772583, 0.3042850196361542, 0.48465654253959656, -0.9920664429664612, 1.2784730195999146, -0.2559778094291687, 0.5533674955368042, 1.2056305408477783, -1.210440993309021, -0.7213240265846252, -1.2066646814346313, 2.2037601470947266, 1.506463646888733, 2.357297658920288, 0.607501745223999, 0.14612098038196564, 0.09685961157083511, 0.8091748356819153, -0.6007703542709351, -1.8384718894958496, -1.5039889812469482, -1.870680332183838, -0.011290977708995342, -0.47630050778388977, 0.34935033321380615, -0.7687957882881165, 1.9764598608016968, -0.16955047845840454, 0.9922041893005371, 1.7860300540924072, -1.1128112077713013, 0.6525231599807739, -1.6573407649993896, -0.9504740834236145, 0.4115927815437317, 0.03397604823112488, 0.27303412556648254, -0.930554986000061, 2.2132887840270996, 0.7748133540153503, -0.32416772842407227, 1.5946483612060547, -1.3871060609817505, 0.5001316666603088, -0.6751644611358643, -0.16716161370277405, -2.1787948608398438, 0.27461346983909607, 0.08319318294525146, 0.5177889466285706, -1.7110283374786377, 0.7989671230316162, -0.21426022052764893, -0.8558897972106934, -0.07507357746362686, -0.4660665690898895, 0.11223573237657547, 0.5399727821350098, 1.7202317714691162, 0.24454215168952942, -0.663680911064148, 0.07973264902830124, -0.881132185459137, -1.7431392669677734, 1.268006443977356, -0.3822450637817383, 0.9085100293159485, -0.5321052670478821, 0.36578884720802307, 1.0763282775878906, -0.5084636807441711, 1.0895426273345947, 0.4793144166469574], [-0.13824012875556946, -0.6236432790756226, 1.8731263875961304, -2.025477409362793, 0.355569064617157, 0.2035660743713379, -0.6309971809387207, -0.36178064346313477, -0.31040266156196594, -1.1994880437850952, -0.7363905310630798, 0.35754120349884033, -0.6984764337539673, -0.5748593807220459, 1.8508220911026, -0.34990808367729187, 0.5595781207084656, -0.5936506390571594, 1.0741791725158691, 0.6672895550727844, -0.6362127661705017, 0.36675605177879333, -0.6322656869888306, 1.7610483169555664, -0.3247397243976593, 1.0139826536178589, -0.7812045812606812, -0.33935293555259705, 1.0438542366027832, 0.17771050333976746, -0.39027491211891174, 0.4785287380218506, -0.804312527179718, 0.26983317732810974, 1.8345102071762085, -1.6961268186569214, 1.112380027770996, 0.034965042024850845, 0.6585492491722107, 0.7700450420379639, 0.18016411364078522, 0.3266383111476898, -0.05000729858875275, -0.6177929639816284, -1.7807917594909668, -0.6573780179023743, -0.10911595821380615, -0.5245906114578247, 0.9912651777267456, 0.14977401494979858, 1.4266059398651123, -1.377277135848999, -2.484408140182495, -0.9422338604927063, -0.39868199825286865, -1.6038225889205933, -0.15676291286945343, -0.5801101922988892, 0.4770190119743347, -0.3146389126777649, -0.9842071533203125, -0.29566606879234314, -0.42142364382743835, 0.479708194732666, 0.19980503618717194, -0.8588428497314453, 1.3093452453613281, 1.1256438493728638, -0.08680176734924316, 0.5076740980148315, 1.596805453300476, 0.3587501645088196, 0.5652275085449219, 0.49876877665519714, -0.42106157541275024, -1.5734223127365112, 0.13860934972763062, -0.969156801700592, 0.2741253972053528, -1.957756757736206, 0.3561290502548218, 0.6779357194900513, 0.6015066504478455, -0.2873084247112274, 1.266445279121399, -0.9361401200294495, 0.5479217767715454, 0.1744958460330963, -1.1635428667068481, 0.9048059582710266, 0.7078237533569336, 0.06072906404733658, -1.3501403331756592, -0.18382839858531952, 0.07436036318540573, -0.6688230037689209, -0.5388957858085632, 0.01681826077401638, -0.4772931635379791, -0.08423267304897308, 1.0118590593338013, 0.21525970101356506, 0.40641242265701294, 0.659523606300354, -0.68939608335495, 0.5290457010269165, 0.40554022789001465, -0.3530423641204834, 1.3578834533691406, 0.2227255403995514, 1.879406213760376, -1.5385719537734985, -0.9347366690635681, -1.2366430759429932, 2.3327295780181885, -1.8353638648986816, 0.03212623670697212, 0.6988247632980347, -0.45648109912872314, -0.6359574198722839, -1.7414757013320923, 0.1483328491449356, -1.9264439344406128, -0.5616359114646912, 0.08385786414146423, 0.12596803903579712, 1.4694262742996216, -1.8955016136169434], [0.006898972671478987, -0.9366902112960815, 0.20628802478313446, -0.18099714815616608, 0.11755242198705673, 2.4528064727783203, -0.13990607857704163, 0.9067209959030151, -1.2096292972564697, -0.2915090024471283, 0.541398286819458, 1.8455814123153687, -0.004531294107437134, 1.3127496242523193, -0.372591495513916, -0.23111163079738617, 1.4330615997314453, 0.17836841940879822, -0.2677774429321289, 1.0299402475357056, 0.5523893237113953, 0.6657067537307739, 0.8561261892318726, 1.390733242034912, -0.1823151856660843, -0.6674240827560425, 0.7243112325668335, -0.34231650829315186, 1.1245841979980469, -0.8360812067985535, 1.1644290685653687, -0.6716720461845398, -0.6877070665359497, 0.6032411456108093, 0.13705477118492126, -0.5896507501602173, -0.11057668179273605, 0.42650070786476135, 1.393757700920105, 1.62144935131073, -0.2365415394306183, 1.3984626531600952, 0.13903096318244934, -0.46684983372688293, -0.17094630002975464, 0.014656594023108482, 0.6097854375839233, -0.7078230381011963, 2.0881214141845703, -0.1174592599272728, 0.0027999808080494404, 0.4303348958492279, -0.07672677934169769, 1.8603382110595703, 0.7948001027107239, 1.0048922300338745, -1.5912433862686157, 0.3272806406021118, 2.5374834537506104, 0.15674304962158203, 1.0554031133651733, -0.8441285490989685, 1.3743410110473633, 0.5000596046447754, 0.5731101036071777, -0.2529686689376831, -0.3785392642021179, 0.9330712556838989, 0.8530306816101074, -0.4317358732223511, -0.20014119148254395, 0.05461704358458519, -0.5837363600730896, 0.3112331032752991, 2.295074462890625, -1.4320553541183472, 1.8806825876235962, 0.7070780992507935, -0.5944324135780334, 1.147108793258667, -0.3666037321090698, -0.7797318696975708, -0.09738321602344513, 1.2456278800964355, 0.14316023886203766, 1.256921410560608, -2.115015745162964, -0.10211780667304993, -0.631485641002655, -1.7752842903137207, -0.06037002429366112, -0.07372613996267319, -0.9572827816009521, -0.6491747498512268, -1.1417702436447144, 1.2810883522033691, 0.8145369291305542, -0.6000768542289734, -0.32409340143203735, -0.7252498269081116, 0.8128786087036133, 0.008547293022274971, -1.8868838548660278, 0.09637649357318878, 0.2536928653717041, 0.20755957067012787, -0.7184447646141052, 1.8983722925186157, 0.5894224643707275, 0.2190379649400711, 0.5097314715385437, 1.600434422492981, 0.32410919666290283, 0.15252822637557983, -0.053728848695755005, -0.6795122623443604, 0.7614744305610657, -0.7239494919776917, -0.5518864393234253, -0.01580028235912323, 0.1180163323879242, -0.06848996877670288, -1.4621796607971191, -0.6572047472000122, -1.6968427896499634, 2.5183768272399902, 1.5263235569000244, 0.10840823501348495], [1.245985746383667, 0.8224339485168457, 0.05023574084043503, 1.034501075744629, -0.13084954023361206, 0.8432925939559937, -0.6328747868537903, 0.06506690382957458, 0.45762205123901367, 2.7074055671691895, 0.06449328362941742, -0.6433879137039185, 1.4464987516403198, -0.6166985034942627, -0.7489277124404907, -0.16184742748737335, -1.5662420988082886, 0.405396044254303, 0.19539572298526764, -0.08725375682115555, -0.385847270488739, -1.1704813241958618, 0.6311340928077698, -0.8475723266601562, -1.5622572898864746, -0.05264921486377716, 0.07353197038173676, -0.34778934717178345, 0.4066813886165619, -0.7802730202674866, -2.093743324279785, -0.311440110206604, -0.7302749752998352, -0.038812268525362015, 0.13144105672836304, 0.3126733601093292, -1.1831791400909424, 1.0610350370407104, -0.5235282182693481, -2.194789409637451, 0.5274530053138733, 0.271406352519989, 1.0112005472183228, 1.9146918058395386, -1.245855450630188, 1.102526068687439, 0.211227148771286, -0.6017172336578369, 1.3232698440551758, -0.24037450551986694, 0.6040043830871582, 0.38325732946395874, -1.1322882175445557, -0.5360875725746155, 1.0922640562057495, -0.1588248461484909, 1.0802483558654785, 0.27522778511047363, -0.7973790168762207, 0.5776055455207825, 0.33989736437797546, 0.8062697649002075, 0.5229179263114929, -0.2896060049533844, -0.7876690030097961, 0.5438280701637268, -1.171398401260376, 0.2469751536846161, -0.030250560492277145, 0.4099124073982239, -0.21170684695243835, 0.3311634957790375, -1.2505276203155518, -0.2180621474981308, 0.8683530688285828, -0.4856693744659424, 0.1914369910955429, -0.6008977293968201, 0.3369409441947937, 0.5295847654342651, 0.09381098300218582, -1.812427043914795, -1.2312076091766357, 0.6775239706039429, 1.8533780574798584, -1.509889841079712, 0.07466947287321091, 0.4580405354499817, 0.8644402623176575, 0.21435044705867767, -0.6015608906745911, -0.5624609589576721, -1.56314218044281, -0.07139945030212402, -0.3010666072368622, 1.6056979894638062, 1.0997339487075806, 1.5921261310577393, -1.1833746433258057, 0.8920708298683167, -0.5290278792381287, -0.6177178621292114, -0.8912904262542725, -1.188346266746521, -1.1103944778442383, 1.3405003547668457, 0.6979144215583801, -2.0195670127868652, 0.09267640113830566, -0.14108099043369293, 0.3068147301673889, 0.7284885048866272, -0.9474674463272095, 0.02387341484427452, -0.2692553699016571, 1.0007479190826416, 1.264705777168274, 2.519178628921509, 0.6059780716896057, 0.5470141768455505, 0.19388189911842346, -0.24700121581554413, 0.1463858187198639, 0.0021647990215569735, -0.4112452566623688, 1.5207958221435547, -0.04418526962399483, 0.1814921647310257], [0.06039983406662941, 0.5148623585700989, 1.0456578731536865, 1.099137783050537, 0.7433632612228394, 0.24162831902503967, 0.8857450485229492, 1.868542194366455, -0.03851870447397232, -0.3277965188026428, -0.7063629627227783, -1.7967133522033691, 1.0848479270935059, 2.2356653213500977, -0.6547008752822876, 0.019033921882510185, 1.7452664375305176, 1.850892186164856, -0.5184300541877747, 0.7483748197555542, 1.418215036392212, -1.3037893772125244, -0.12275150418281555, -0.12072785943746567, 0.02007768675684929, -0.46967488527297974, 0.18859261274337769, -1.1826571226119995, 0.26646649837493896, -1.3857370615005493, -1.4432806968688965, 0.567203164100647, 0.020060909911990166, 0.911746084690094, -0.13164447247982025, -1.6276947259902954, 0.40445488691329956, 0.5387449860572815, -1.5447883605957031, -0.15032435953617096, -0.28921395540237427, 0.21640101075172424, 1.7883445024490356, 0.9819422960281372, 0.7836242318153381, 0.17447875440120697, -0.8577305674552917, -0.04973727464675903, 0.8522387146949768, -1.3389168977737427, 0.1250825822353363, 1.5267791748046875, -1.062552809715271, -1.7063333988189697, 0.4202330410480499, -1.0583380460739136, 2.6116440296173096, -0.6345529556274414, -1.0997051000595093, -0.3025435209274292, 0.18421560525894165, -1.0314501523971558, 1.2979514598846436, 0.9590157270431519, -0.9728499054908752, -1.1474848985671997, 0.5906428694725037, 0.13540448248386383, -0.5510192513465881, -1.0546605587005615, -0.5312573313713074, -0.039294783025979996, 1.022036075592041, 0.04745658487081528, 0.02622557058930397, 0.35364001989364624, -1.17306387424469, 0.1327899992465973, -0.1317264288663864, 0.6003299355506897, 0.8896791934967041, -0.47333189845085144, 1.1429494619369507, 0.2786897122859955, -0.3442441523075104, 0.6840417385101318, -1.0288374423980713, -1.264035940170288, 0.8500348329544067, 1.277319073677063, 0.6613701581954956, 0.22564542293548584, -1.660696268081665, -0.8202853798866272, 0.5508791208267212, 0.8607025146484375, 1.7400144338607788, 0.40343770384788513, 1.5060678720474243, -2.562649965286255, -0.6559869647026062, 0.09278348833322525, 2.393617868423462, -0.9300442934036255, 0.6751897931098938, -0.7417855858802795, -0.24584992229938507, 0.49309223890304565, -1.2133581638336182, 0.3914440870285034, 0.8134040236473083, 1.2065351009368896, -1.3668413162231445, 1.8092799186706543, 0.31529149413108826, -0.18301557004451752, -0.1270982176065445, -1.5778729915618896, -1.8347840309143066, 1.733243703842163, -0.10927610099315643, 0.18889394402503967, -0.7218779921531677, 0.5355440974235535, 0.27428048849105835, -0.10125922411680222, 2.9176156520843506, 0.019780900329351425], [-0.36197227239608765, 0.2759121358394623, 0.41499876976013184, -0.154571071267128, -0.33909091353416443, 0.8731769919395447, -2.634822368621826, -0.21472755074501038, 1.7628237009048462, -1.279331088066101, 0.06510278582572937, 0.9006224870681763, -0.23839832842350006, -0.056590355932712555, 0.6440843343734741, -0.5800577998161316, 2.231891632080078, -0.19217872619628906, -1.1038856506347656, 1.456201195716858, 0.08215991407632828, -0.19070833921432495, -1.4613205194473267, 0.04697893559932709, -0.5629817247390747, 2.777038812637329, -1.147079586982727, -0.7275455594062805, 1.165299415588379, 0.4550614655017853, -0.5599344372749329, -0.5889617204666138, 0.4726971685886383, -0.10705451667308807, 0.2724737823009491, 1.147363543510437, 1.564773678779602, 0.03911396488547325, 2.1471822261810303, 1.1172034740447998, 0.9522812366485596, -0.29050201177597046, 0.052594710141420364, 0.010129814967513084, 1.0728024244308472, -0.3285391926765442, -1.1279585361480713, -0.011100678704679012, -1.1654969453811646, -0.679211437702179, 1.7682812213897705, 1.6758594512939453, 1.5168861150741577, 0.44880378246307373, 0.4884052574634552, 2.104816198348999, -0.11107838898897171, -0.7228589057922363, 0.5466212630271912, 0.9627138376235962, 0.3987405300140381, 0.7530248165130615, -0.066083624958992, -0.04705885052680969, 1.5113990306854248, -0.9992655515670776, 0.01881979964673519, 0.6638233065605164, 1.7171893119812012, 1.1661298274993896, 0.0849640741944313, 0.4647621512413025, 0.3870787024497986, -1.1251555681228638, -0.5234542489051819, -1.3299020528793335, 1.1513651609420776, 1.6093932390213013, -1.8787341117858887, -0.6420812606811523, -0.32379984855651855, 0.44057774543762207, 1.1365245580673218, -0.15803393721580505, -0.014292122796177864, 0.15189355611801147, -0.5544903874397278, -0.5762672424316406, -2.0339508056640625, -0.42036527395248413, -1.254637360572815, -0.8038718104362488, -0.17653660476207733, 1.6306942701339722, 0.2520360052585602, -1.4017672538757324, -0.9876916408538818, -0.4415554106235504, -0.24509818851947784, 0.08164471387863159, -0.6200768351554871, -1.300667643547058, -0.7318315505981445, 0.13236366212368011, 1.3340210914611816, 0.14709687232971191, 1.6227043867111206, 0.7539213299751282, -0.12825830280780792, 0.4745849668979645, 0.9102305173873901, -1.6236577033996582, 1.1353610754013062, 0.5934917330741882, -0.017902350053191185, 0.36290106177330017, 0.7628192901611328, -0.7017068862915039, 1.8113847970962524, -1.1395909786224365, 0.0009004785097204149, -0.798536479473114, -1.2407511472702026, -0.36424434185028076, -0.665532112121582, -0.4169163405895233, 0.10257195681333542, -0.5208840370178223], [0.37481239438056946, 0.7393807768821716, -0.05060875043272972, -1.0409053564071655, -0.02635345421731472, 1.0427225828170776, 0.48824959993362427, 0.5366032719612122, -0.08529666066169739, -0.8509952425956726, -0.42698121070861816, -0.8468114733695984, 1.3812191486358643, -0.47021302580833435, 0.46192023158073425, -0.8212835192680359, -1.73673677444458, -1.9333393573760986, -0.5932973027229309, -0.9568783044815063, -0.5269362330436707, -0.9243955016136169, -0.24651378393173218, -1.1261836290359497, -1.253774881362915, -0.8706455230712891, 0.16892388463020325, -0.4157739579677582, -0.2372763305902481, -0.3138226568698883, 2.6513540744781494, -1.8019616603851318, 0.8157463669776917, -0.6599823832511902, -1.17979097366333, -0.9896227121353149, -1.9554367065429688, 1.3398247957229614, 1.89993417263031, -1.288777232170105, 0.13961409032344818, -0.9564326405525208, 1.0583367347717285, -0.4579560160636902, 1.6659348011016846, -0.8351659774780273, -0.5019871592521667, 1.5497994422912598, -0.4866016209125519, 0.5928120613098145, 0.1851060688495636, 0.660219132900238, 0.7527068853378296, 1.0626918077468872, -0.2784678041934967, -0.27391496300697327, 0.7548210024833679, 0.6359609961509705, 1.3936121463775635, 0.13262934982776642, 0.17187772691249847, 1.805724024772644, 0.4007536768913269, 0.6064054369926453, 0.7365120649337769, -0.6184512972831726, 0.3370940089225769, 0.6245577335357666, 0.488386869430542, -1.360787034034729, -0.23865002393722534, -0.21685858070850372, -0.8407602906227112, 0.8654202222824097, -0.7519369125366211, -1.096961259841919, 0.3566941022872925, -0.7320011854171753, 0.39859291911125183, 0.8338048458099365, -0.34885331988334656, -0.22338831424713135, -0.040411576628685, 2.354116678237915, -0.13221020996570587, -1.7909711599349976, 0.9939929246902466, -0.4421074092388153, -1.863772988319397, 0.6390257477760315, 0.372078001499176, 0.09067989885807037, -0.3413582444190979, -1.8003407716751099, 0.2497854232788086, -0.09198721498250961, -1.6167372465133667, -0.5353022813796997, -1.2009738683700562, -1.3447474241256714, -0.5067072510719299, 1.0644696950912476, -0.30521270632743835, 0.06831859797239304, -1.8480690717697144, 1.7644410133361816, 0.15802377462387085, 0.21121160686016083, -0.25156980752944946, -0.03367907926440239, 0.1335606873035431, 0.31092920899391174, 1.0115066766738892, 0.7935584187507629, 0.4385035037994385, 0.516726553440094, 2.7586371898651123, 0.6874586343765259, 0.30090200901031494, -0.28592321276664734, -1.737239956855774, 1.2696831226348877, -1.1544678211212158, -0.3310231566429138, -0.5723614692687988, 0.5935766100883484, -0.13701894879341125, 1.298587441444397], [1.0141785144805908, 0.2787356376647949, 0.5234174728393555, -0.9516198039054871, 0.533342719078064, 0.14799357950687408, -0.4883797764778137, -2.3833799362182617, 0.2314143031835556, -0.9690670371055603, -1.1491577625274658, -0.77333664894104, -0.20374104380607605, 2.109168291091919, -0.3168613314628601, 0.9081152081489563, -0.7126338481903076, -0.4184395968914032, 0.4457177221775055, -1.06803560256958, -0.5594552755355835, 0.6817177534103394, 2.9613921642303467, -0.7211971879005432, -1.0497087240219116, 0.8461174368858337, 1.4062148332595825, -1.0280896425247192, 0.1344779133796692, 0.22752277553081512, 1.8590692281723022, 0.990121603012085, -0.19428937137126923, 0.18271076679229736, 0.4901944696903229, -0.5091000199317932, 0.8898229598999023, -0.664966881275177, 0.9726380705833435, 0.28089192509651184, -0.4317418038845062, 0.5630913376808167, 0.34261757135391235, 1.7892850637435913, 1.5387741327285767, -0.6043345332145691, -1.0667792558670044, -1.6058191061019897, 0.39213404059410095, -1.7242720127105713, -1.162481665611267, 0.41126981377601624, 2.161736249923706, 0.6517265439033508, -0.5247413516044617, -1.2915877103805542, 0.320585161447525, -0.6063783168792725, -0.5031176805496216, 1.399290680885315, -0.12036089599132538, 0.865421712398529, -0.04858551174402237, -0.3918607532978058, 0.5860113501548767, 1.5402458906173706, -1.6392422914505005, -0.4553600251674652, -1.2230215072631836, 1.4597692489624023, 1.5239932537078857, 0.6539998054504395, 0.48196446895599365, 0.6748185157775879, -0.31831085681915283, 0.5381873250007629, 0.39566904306411743, -2.101543664932251, -0.1940605193376541, 0.16615454852581024, -1.066212773323059, -0.28157082200050354, 0.19321618974208832, -0.924919068813324, 0.29884424805641174, -0.5573627948760986, -0.1989843100309372, 1.3357704877853394, -1.1214873790740967, -0.5228282809257507, -0.4813956916332245, 0.697656512260437, -0.06342082470655441, 1.7162622213363647, -0.07758072018623352, 0.4413641393184662, 0.37365564703941345, 0.5417723059654236, -1.2752434015274048, 1.3914977312088013, 1.4225118160247803, -2.003232479095459, 1.6168816089630127, 0.574565052986145, 1.2611204385757446, -1.0410748720169067, -0.7717952132225037, -0.048692092299461365, -0.2164325714111328, 0.49355876445770264, -0.5267854332923889, -0.10942982137203217, 1.2463160753250122, -1.477711796760559, 0.7086997628211975, 0.2156875729560852, 1.0004689693450928, 0.8106789588928223, 0.38990819454193115, 0.3499144911766052, -0.1526554822921753, -0.9644506573677063, 2.4904863834381104, 0.7154467701911926, 0.21346713602542877, -0.4163859188556671, -0.15031099319458008, 0.2965020537376404], [-0.5773682594299316, -2.65643310546875, -0.6503128409385681, 1.1952029466629028, -0.751301109790802, -0.6199477910995483, -0.7404523491859436, -0.7617596387863159, 0.9394859075546265, -0.15530799329280853, -0.002965510357171297, -0.6641995310783386, 1.4652396440505981, -1.0815187692642212, -1.2729700803756714, 2.6432909965515137, 0.1257120817899704, 0.4059220254421234, 1.843742847442627, 0.17304295301437378, 0.9515150785446167, 0.4659690260887146, -1.3465440273284912, -1.0475285053253174, -1.1035641431808472, 0.9730293154716492, 0.4451615810394287, 1.6345088481903076, 0.20750795304775238, -0.4459620714187622, 0.20537284016609192, -0.9691748023033142, -0.9403908252716064, 0.6864529252052307, -0.5674440264701843, -0.8354660868644714, 0.5238839983940125, 0.8751201033592224, -0.22180229425430298, 0.3981810510158539, 0.42276257276535034, -1.7439817190170288, -0.1665111780166626, -0.1513587236404419, -1.077311396598816, 1.463586688041687, 0.2097272425889969, 2.501906156539917, -0.331900030374527, -0.4597042202949524, -0.7195814847946167, 1.1068884134292603, 0.887180745601654, -1.2013887166976929, 0.261190801858902, 0.5171136260032654, -0.06029545143246651, 0.8608894944190979, -2.840999126434326, 0.6582090258598328, -0.22182327508926392, -0.218641996383667, -1.0281931161880493, -0.5967628955841064, 0.4307977259159088, 0.7444051504135132, 1.5170140266418457, 0.09729678928852081, -0.8569130897521973, 0.7917195558547974, 0.3431234061717987, 0.5144625902175903, 1.8546786308288574, 1.0726442337036133, -1.082294225692749, -0.3388760983943939, -1.4648804664611816, -1.0785995721817017, -0.6137170195579529, 0.8901976346969604, 1.3949495553970337, 1.4139435291290283, -1.4066777229309082, 0.4255523085594177, -0.9777214527130127, -0.3547845482826233, 0.9616885781288147, 1.1528735160827637, 1.4386707544326782, -0.6459298729896545, 0.14588718116283417, -0.8591387271881104, -0.6852079629898071, -1.1919842958450317, 0.2071334570646286, -0.8147879838943481, -0.4943852722644806, 0.6518956422805786, -0.4077717065811157, -1.1327778100967407, -0.47576704621315, 0.7954860329627991, 0.08262189477682114, 0.6508235335350037, 0.3350944519042969, 0.3411354720592499, -0.2484237551689148, -0.9950767159461975, 1.5196539163589478, 2.213515043258667, 0.8288090825080872, 1.7501837015151978, -0.5170435905456543, 1.3725051879882812, -1.5215637683868408, -0.406000554561615, 0.2599843144416809, 0.09801970422267914, -1.195818543434143, 1.1106661558151245, -0.012055573053658009, -0.8910885453224182, 0.6860653758049011, -2.58782696723938, 1.9072877168655396, -1.527959942817688, 0.43956705927848816, 0.7910876870155334], [2.6190073490142822, -0.6345551013946533, 0.15654881298542023, 0.922415018081665, -0.9567162990570068, 1.2958672046661377, -0.206191286444664, -0.9781954884529114, -0.7321685552597046, 0.7007618546485901, 0.2650130093097687, 0.9841886162757874, -0.6976900100708008, 0.7477232813835144, 1.0279879570007324, 0.8159054517745972, 0.8773148059844971, 1.3243699073791504, -1.1803303956985474, 0.022058960050344467, -0.9551184177398682, 0.8229681849479675, 0.6485821604728699, -0.7579406499862671, -1.0899486541748047, -0.268798291683197, -0.7103654146194458, -0.2561142146587372, 1.71177339553833, 1.4681081771850586, 1.3059868812561035, 0.32699891924858093, -0.3882179856300354, -0.5969104170799255, -0.13271042704582214, 0.09402555227279663, 0.19680626690387726, -1.5222115516662598, 1.0000815391540527, 0.6550776958465576, 1.916835904121399, 0.7101430296897888, -0.3790178894996643, -0.3042687773704529, 0.8932454586029053, -1.2662856578826904, -0.9067181348800659, 1.4080584049224854, -1.4324244260787964, 0.7922170758247375, 1.4961482286453247, 1.1896401643753052, -1.2635178565979004, -0.05226491764187813, 0.5525719523429871, -1.04067063331604, -0.9263971447944641, -0.6295642852783203, 0.024678636342287064, -0.7096270322799683, -0.4618966281414032, 0.2632853388786316, -0.9738096594810486, -0.36775335669517517, -0.08415339887142181, -0.40714165568351746, 0.2768445611000061, 1.1314373016357422, -0.21371987462043762, 0.3669006824493408, -0.2084420621395111, 1.4437899589538574, -0.7148658037185669, 0.2934757471084595, 0.4946877062320709, -0.274130254983902, -0.5838860869407654, 0.5480419993400574, -0.29088833928108215, 0.8654425740242004, 0.46720901131629944, 0.8969773054122925, -1.0112656354904175, 1.4959264993667603, -1.2581653594970703, 0.5900666117668152, 0.7773799896240234, -1.4477660655975342, -0.47484204173088074, -2.8962388038635254, -0.8327948451042175, -1.0361100435256958, 0.7838690876960754, -0.7907845973968506, 0.33273911476135254, -0.20115713775157928, -0.26952627301216125, 0.9657031893730164, 0.9203658699989319, -0.296775758266449, 0.3130238652229309, -0.3106272518634796, -0.513761043548584, -1.627547264099121, 0.670407235622406, -1.0103733539581299, -1.2799599170684814, 0.43013995885849, 0.8859147429466248, 0.4393448233604431, -1.580958604812622, -1.1154699325561523, -0.7988765239715576, -0.2745530605316162, -1.3556857109069824, 0.2273760586977005, 0.20681911706924438, 1.595005750656128, 0.3931671977043152, -0.6078600883483887, 1.6358528137207031, -0.06837828457355499, 0.04897712171077728, 2.1959869861602783, 2.2080490589141846, 0.32007384300231934, 0.2296435385942459, -0.4343728721141815], [0.10660973191261292, 0.20620012283325195, -1.233215093612671, -0.1302013099193573, 1.2637335062026978, 0.18380682170391083, -0.6394187808036804, -0.3316882848739624, -0.6234696507453918, 0.1347484290599823, 2.055614709854126, 0.24959509074687958, -0.3525085747241974, -0.3493722081184387, 0.16172273457050323, 0.24206672608852386, 0.7415526509284973, -0.5489122271537781, -1.6071077585220337, 1.9382423162460327, 0.10216047614812851, 0.2723945379257202, -0.4680514335632324, -1.7000232934951782, -0.3049066662788391, -0.3164631724357605, -0.42860254645347595, 0.6193745136260986, -1.3574233055114746, 0.1679830104112625, -0.7340013384819031, -0.7819495797157288, -0.7485335469245911, 0.3154226839542389, -2.031698703765869, -0.17389875650405884, 0.924102783203125, 0.3554118871688843, 0.2643640339374542, -0.7782427668571472, -0.682273805141449, -0.14106062054634094, -0.21886493265628815, 0.6995849609375, -0.014027396216988564, 1.3579065799713135, 0.00012810285261366516, -0.9771595597267151, -0.26404815912246704, 1.5756914615631104, 1.3666293621063232, -0.048949986696243286, -1.225791335105896, -1.7706397771835327, 0.33328261971473694, -1.0166499614715576, 1.167086124420166, -0.2323908805847168, 0.12028958648443222, 0.39707323908805847, 0.1669100821018219, 1.0314463376998901, 1.472501277923584, -0.30509284138679504, 0.6810784339904785, -1.0480551719665527, -1.2050987482070923, 0.25020965933799744, 0.7350504994392395, 0.703513503074646, -1.9035753011703491, 0.5846598148345947, -0.030452866107225418, 0.36804044246673584, 0.4039076268672943, 0.3248462975025177, -0.299143522977829, -1.2462011575698853, 0.672847330570221, 0.2167336344718933, 0.010553599335253239, 0.05714103952050209, -1.468438982963562, 0.11427493393421173, 1.2439990043640137, -1.2744526863098145, -0.6570391654968262, 1.4090971946716309, 0.7306652069091797, -2.130199670791626, -0.1574118584394455, 1.0264228582382202, -0.923440158367157, -2.1052746772766113, 1.1238148212432861, -1.5338870286941528, -0.913249671459198, 0.38110464811325073, -0.5671474933624268, -1.927554965019226, -0.13884177803993225, 1.0471547842025757, -0.2089315950870514, 0.4571596384048462, 0.9952155947685242, 2.148427963256836, -0.5572170615196228, -1.5287835597991943, 1.4036736488342285, -0.3620815575122833, -2.600881576538086, 0.17947173118591309, 0.47111427783966064, -1.214441180229187, 1.0333019495010376, 1.2356728315353394, 0.9033457636833191, 0.785673201084137, 0.8408862948417664, -0.005804033949971199, 0.5783060193061829, 0.39478591084480286, -0.7601401805877686, 0.04594140127301216, -0.8249189257621765, 0.03520258888602257, -1.0560401678085327, -0.10986831039190292], [-1.1801735162734985, 1.3862175941467285, -2.3329389095306396, 0.7026486396789551, 0.6470919251441956, -1.6296234130859375, -1.3636178970336914, 0.8192855715751648, -0.06802505999803543, 0.2702970802783966, -0.5901293754577637, -1.3637357950210571, -1.7642707824707031, -1.1698195934295654, -0.4364323914051056, 0.10011555254459381, 0.23045074939727783, 0.18118810653686523, 1.4653433561325073, 2.1835334300994873, -1.0576698780059814, 0.04850425198674202, 0.926899254322052, 0.26515161991119385, 0.3048958480358124, 1.4868000745773315, 0.018100667744874954, 1.9353545904159546, -1.1087255477905273, -1.1541825532913208, 0.678111732006073, -0.5210803747177124, -1.018410563468933, 0.08152899891138077, -0.2600734829902649, -0.0992702767252922, -0.5651049613952637, 0.6413083672523499, 0.4133710265159607, 0.28467419743537903, 0.4852955937385559, 0.3486042022705078, 0.6368513107299805, 0.14856494963169098, -2.2078235149383545, 0.5144214034080505, 1.7596970796585083, -3.0397238731384277, -0.35413962602615356, -0.2796511948108673, -0.982568085193634, 1.4254803657531738, -0.4388135075569153, 0.6242180466651917, -0.6227752566337585, 0.6310142278671265, 0.0805584043264389, -0.8545986413955688, -2.0797057151794434, 0.15642452239990234, -0.2322118580341339, 1.5944883823394775, 0.5369894504547119, 0.24786871671676636, 0.09833991527557373, -0.8531870245933533, 0.926268994808197, 0.7687948942184448, 1.4340791702270508, 2.6096527576446533, 0.5160561800003052, -0.27964481711387634, 0.16903150081634521, 0.2865994870662689, -0.9641364216804504, -2.018389940261841, 1.0985503196716309, 0.012644958682358265, -0.9470762610435486, -0.9035395979881287, -0.10257662832736969, 0.8667472004890442, -0.4372125267982483, -0.8087064623832703, 0.20981548726558685, 0.4316079318523407, -0.2413109689950943, -0.3865751624107361, 0.165854811668396, -1.3651388883590698, -0.06544531881809235, -0.7799137234687805, -1.5902775526046753, -0.9249698519706726, -0.21242350339889526, 0.4019272029399872, 1.7423503398895264, -0.9646853804588318, -0.6301887035369873, -0.3715992867946625, -1.1469392776489258, -0.08131330460309982, -0.21142180263996124, 0.248740091919899, -0.4637015163898468, 0.2924555242061615, -0.5539728999137878, 0.5495312809944153, 0.9333049654960632, 1.043796181678772, -1.7258473634719849, 0.6918318867683411, -0.6675514578819275, -0.3630768358707428, -1.588450312614441, 2.3255531787872314, 0.5291022658348083, 0.9481614828109741, -1.373457908630371, -3.2185449600219727, 0.713922917842865, 1.147174596786499, 0.3087593913078308, 2.580979108810425, 0.2691936492919922, -0.6284129023551941, -1.6456528902053833, 0.848137378692627]]], [[[0.14793457090854645, -0.633588969707489, 0.12321283668279648, 0.9565592408180237, 1.8608301877975464, -0.32262659072875977, -0.8997958898544312, 0.5904977321624756, 1.6272974014282227, 0.02249693125486374, 1.5319480895996094, 0.18898913264274597, 1.6118346452713013, 1.0354769229888916, 1.1200318336486816, 0.857553243637085, 0.20291778445243835, -0.6717191934585571, -1.256550908088684, 0.46053943037986755, 0.09872328490018845, 0.5613888502120972, -0.8136757016181946, -0.7240001559257507, 1.8337594270706177, -1.2765815258026123, 1.5460336208343506, 1.4169145822525024, 0.6859226822853088, -0.21410219371318817, -1.6306803226470947, -1.6482253074645996, 1.3330192565917969, -0.49765506386756897, 1.6768172979354858, -0.2778354287147522, -1.724382996559143, -0.21283844113349915, -1.28813898563385, 1.01883864402771, 0.2074287384748459, -1.0786558389663696, 2.348965644836426, -0.43090713024139404, -0.36655887961387634, 0.5552963614463806, 1.540306806564331, 0.48236095905303955, -1.0615895986557007, 1.1511218547821045, 2.048722982406616, -0.46006688475608826, 1.1497470140457153, 0.4357863664627075, 1.0653787851333618, -0.5039926767349243, 0.4744580090045929, -0.046421751379966736, -0.6189548373222351, 0.24998725950717926, 0.21443292498588562, -1.0407925844192505, 1.195282220840454, -1.6665925979614258, 0.3913528621196747, -0.9499136805534363, -1.424364686012268, 0.11201810091733932, 1.492392659187317, -1.108574390411377, -0.02943381480872631, 0.01547225657850504, -0.9962319731712341, 1.1185885667800903, 1.8052352666854858, 0.49814045429229736, 2.259744882583618, 1.4174258708953857, 0.8011571764945984, 0.4795185327529907, 0.89701908826828, 0.34081751108169556, 0.6415333151817322, -1.1948856115341187, -0.425226092338562, 0.11657311022281647, -0.15437886118888855, -3.0620532035827637, -0.4007043242454529, 1.1173474788665771, -1.3315248489379883, 1.2379456758499146, 0.6288782358169556, -0.738993227481842, 1.2607805728912354, 0.27742481231689453, -0.5227211117744446, -0.2387782633304596, 1.3135846853256226, 0.3454268276691437, -0.22661663591861725, -0.5969099402427673, -1.0926653146743774, -0.41232264041900635, -0.7424405217170715, -0.5981184244155884, -1.1831098794937134, 0.08741626143455505, 0.8154475688934326, 2.041370391845703, 1.2765159606933594, -0.725254237651825, -0.5549282431602478, -1.1208770275115967, -0.9236291646957397, -0.41966503858566284, -0.5945371985435486, -0.2904913127422333, 0.5637875199317932, 0.20499631762504578, -0.5627228021621704, 0.3298323154449463, 0.9931952953338623, -0.8835201263427734, -0.26123642921447754, 1.3210862874984741, -0.5961286425590515, -0.17492204904556274], [0.9816085696220398, -0.5271129608154297, 0.4335893392562866, -1.6074280738830566, -1.7814273834228516, 1.0781493186950684, -0.08418775349855423, 0.3743695318698883, 0.26364412903785706, 0.8396492600440979, 1.22954261302948, 1.2713054418563843, 0.7532607913017273, -0.2538449168205261, -1.1431560516357422, 1.5556740760803223, -0.25561967492103577, -0.7326390147209167, 1.5828338861465454, -0.08407402783632278, 0.5516316294670105, 0.16067533195018768, -0.5243122577667236, -0.12907294929027557, -0.5307212471961975, 0.7498432993888855, 0.60356605052948, -0.2765519618988037, 0.6630931496620178, 0.013684921897947788, -0.8465440273284912, -1.8387705087661743, -0.6129588484764099, -0.5017121434211731, 0.09003148972988129, 0.5938981175422668, 0.7318162322044373, 0.16373994946479797, -0.5568187832832336, -1.0191479921340942, -0.716454803943634, -1.8345451354980469, 1.3187657594680786, -1.819589376449585, 0.33262890577316284, 0.9897860884666443, 2.109612464904785, -1.540290117263794, -1.093387246131897, 1.9906392097473145, -0.1307927817106247, -0.11778603494167328, -1.1110739707946777, -0.7718385457992554, 1.1548070907592773, 0.5629271864891052, 0.6915026903152466, -0.13103406131267548, 0.28843969106674194, 0.8702535629272461, -0.039877742528915405, -0.4614766538143158, 1.068325400352478, -0.37409061193466187, 0.723721444606781, -1.247727394104004, -0.1095585897564888, 0.8601913452148438, 1.0132390260696411, 2.0568528175354004, -1.7022075653076172, 1.3760730028152466, -1.2802482843399048, -0.9052299857139587, -1.3089429140090942, -0.2681325674057007, -0.7253541946411133, 0.3171413540840149, 1.5846885442733765, 0.16607387363910675, 0.7705948948860168, 1.3164544105529785, -1.1373310089111328, 1.2093241214752197, -0.3934611976146698, 0.4263542890548706, -0.0028579742647707462, 2.493727445602417, -2.6943717002868652, 0.38280603289604187, -1.1178945302963257, 0.3951108455657959, 0.02042256109416485, -0.5331034064292908, 1.128534197807312, 0.37426015734672546, -0.16220679879188538, 0.22240734100341797, -0.38541606068611145, -0.521390438079834, -0.36997753381729126, -0.5325189232826233, 1.2281508445739746, 0.3703407049179077, -0.22426575422286987, 0.3306189775466919, 1.8667854070663452, -0.493432879447937, -0.15662530064582825, -0.11496659368276596, 0.9645814895629883, 0.24014735221862793, 0.4369887411594391, -0.16533023118972778, 0.4244564175605774, 0.5614476799964905, -0.8729563355445862, -0.006227841600775719, 1.746677279472351, -0.10749775171279907, 1.7037763595581055, -0.49592167139053345, -0.853839099407196, -0.26071980595588684, -1.4312069416046143, -0.3393156826496124, 1.1903876066207886, 0.7305769324302673], [0.7111939787864685, 0.07614380866289139, -0.1631489396095276, -1.2711373567581177, -1.7096482515335083, -0.9508009552955627, 0.26282912492752075, 0.012197992764413357, -0.026538683101534843, -1.3608722686767578, 1.5738447904586792, -1.8208534717559814, -0.08516889065504074, -2.9352550506591797, -0.7978156805038452, -1.9331674575805664, 0.38339921832084656, -0.4670446813106537, -1.1542245149612427, 0.5177279710769653, -0.7660685181617737, -0.7324713468551636, -0.9488295912742615, -0.5539054274559021, 1.624585747718811, 0.522257387638092, 1.5402588844299316, 1.4547770023345947, 0.812114417552948, -0.4446168541908264, -0.940485417842865, 1.062940001487732, -0.9031261205673218, -1.560259222984314, -0.7601631283760071, 0.6925244331359863, -0.22335846722126007, 1.0975478887557983, -0.3316269516944885, 0.4766092896461487, -1.3726047277450562, -1.5482800006866455, 0.16737394034862518, 1.15445876121521, -0.1358071267604828, 1.3694239854812622, 0.21384024620056152, 1.3818382024765015, 0.953110933303833, 0.7545493841171265, -0.1889282763004303, 1.6643694639205933, -1.4223906993865967, -0.5349518656730652, 0.15997059643268585, 0.284402072429657, -1.7667235136032104, -1.092879056930542, -0.9805040955543518, 0.7062609791755676, 0.2392130047082901, 0.7252724766731262, -0.7423120141029358, -1.336542010307312, 1.3239706754684448, -0.06756402552127838, -0.8920086622238159, 0.9365264177322388, 0.012876040302217007, 0.489056259393692, 0.46766331791877747, 0.012490482069551945, -1.3815265893936157, 1.9963176250457764, 0.6993412375450134, -1.5095967054367065, -1.1883430480957031, 0.744810163974762, -1.657027006149292, -0.7917162775993347, -0.9939121603965759, -0.959362268447876, -0.8480126857757568, 0.1627279371023178, 0.1585720032453537, 0.6698610186576843, -0.46400779485702515, 0.9180149435997009, -0.9179354906082153, -1.8205851316452026, 0.45082131028175354, -0.5880981087684631, -1.586516261100769, -1.5280861854553223, 0.8807098269462585, 1.7919448614120483, 1.5614160299301147, -0.9514268040657043, 1.5858092308044434, 0.14097540080547333, -0.20838238298892975, -0.12034260481595993, 1.4535640478134155, -1.4573333263397217, 2.3922548294067383, -1.5473383665084839, -0.5834100842475891, -0.3614451289176941, -0.23322223126888275, 1.3143383264541626, 0.8606141805648804, 1.6898155212402344, 1.31832754611969, 2.6290717124938965, 0.9344195127487183, -0.15982714295387268, -0.43363264203071594, 0.6573109030723572, -0.5888703465461731, -1.4435209035873413, 0.15409186482429504, -0.4261621832847595, 1.044999122619629, 0.6567505598068237, 0.32228079438209534, 0.8799705505371094, 0.058988504111766815, 0.5650275349617004], [-1.2130275964736938, -0.7960376143455505, 0.45034441351890564, -0.549869954586029, 1.6473604440689087, -0.9232761263847351, -0.12879124283790588, 0.4981548488140106, 0.7078630924224854, 0.23457299172878265, 1.4531584978103638, -0.37132528424263, -1.6341371536254883, 1.2339985370635986, 0.7763276696205139, 0.5421426892280579, -1.5516045093536377, -1.5676250457763672, -0.8447914123535156, 0.9536464214324951, -0.4926704168319702, 0.13897229731082916, 1.5009719133377075, -0.7868617177009583, 0.20316603779792786, -0.2116231918334961, -0.11411003023386002, -0.4594379663467407, -0.7223487496376038, 1.877820372581482, 0.30892878770828247, -2.4843649864196777, 1.0626615285873413, -0.8291293382644653, -0.9910612106323242, -0.02673276513814926, -0.7933704853057861, -1.4951977729797363, -1.4888702630996704, -0.000423264253186062, -1.2813822031021118, 0.6883595585823059, 0.058286942541599274, 0.6537775993347168, 0.786281168460846, -0.09200965613126755, -0.982719898223877, 0.36987173557281494, -1.079488754272461, -0.9583126306533813, -0.7499823570251465, -0.15775325894355774, 0.24252203106880188, -0.8132706880569458, -3.7730441093444824, -0.45877280831336975, -0.08690951019525528, -1.4421794414520264, -0.6104778051376343, -0.2340753823518753, 0.40943196415901184, -0.3880762457847595, -0.19457389414310455, 0.6275407671928406, 0.43134891986846924, -1.883959412574768, -0.3909747004508972, 0.5426942706108093, -1.5266193151474, -0.1745263636112213, 1.07944917678833, 0.5372724533081055, -0.6537395119667053, -0.37710586190223694, 0.1396797150373459, 1.8707796335220337, 0.23260045051574707, -0.3643359839916229, 1.7138310670852661, -0.5238508582115173, -0.9674363732337952, -2.11651873588562, 0.9732263088226318, 0.3298352062702179, -0.6911611557006836, 0.1490582823753357, -0.14456087350845337, -0.7605840563774109, -1.2502388954162598, 3.18911075592041, 0.1712087243795395, -0.5416553616523743, -1.0020196437835693, -1.5560405254364014, -0.16777877509593964, -0.13238359987735748, 0.30035650730133057, -0.02212369255721569, 1.433889389038086, 1.4188575744628906, -0.8770684599876404, 0.17195086181163788, -0.2116793841123581, -0.16554276645183563, 0.8407093286514282, -0.5936775803565979, -0.14873485267162323, 1.1889495849609375, -1.664837121963501, -0.2330765426158905, -0.6181179285049438, 1.7449541091918945, -1.2295827865600586, 1.5729596614837646, -0.1710413694381714, 0.8633202314376831, -0.10557961463928223, 0.08932037651538849, 0.3330734670162201, 0.8581804037094116, 0.797077476978302, -0.3092099726200104, 0.33875060081481934, -0.7401282787322998, 0.8947504758834839, -0.04918316751718521, -1.1237988471984863, 0.5323474407196045], [0.5416625142097473, -0.24922974407672882, -0.372859388589859, 1.7322096824645996, 0.28234952688217163, 2.154215097427368, -0.3637068569660187, 0.7473267912864685, -0.07955485582351685, 0.09997262805700302, -0.4218815267086029, 0.7116720676422119, 1.268849492073059, 1.2311607599258423, -0.7206701636314392, 0.9038335084915161, 0.8027087450027466, 0.45225733518600464, 0.6127714514732361, 0.3831543028354645, 0.20676888525485992, 0.11903472989797592, -0.5478797554969788, -0.404145210981369, -0.5134187936782837, -0.2345769703388214, -1.34102463722229, -1.2766810655593872, -0.5828160643577576, -0.8784600496292114, -0.11688201874494553, -2.1219899654388428, 0.23190587759017944, -0.18755009770393372, -1.1648058891296387, 0.15401042997837067, 1.0666371583938599, -1.1262511014938354, -2.776740074157715, -0.49642857909202576, -0.9126628041267395, -0.35277923941612244, -0.8789111375808716, 0.9535889029502869, 0.864729106426239, 0.8080670237541199, -0.12523002922534943, 3.200011730194092, -1.7393486499786377, -1.3407049179077148, 0.11233891546726227, 0.5567641854286194, 0.04729501157999039, -0.5343733429908752, 1.3090988397598267, -0.619529664516449, -0.787208616733551, 0.6838204264640808, 0.9765356183052063, 0.10677660256624222, 0.009250590577721596, 0.09886720031499863, 0.9485875368118286, -0.06232141703367233, -0.29264527559280396, 0.9163771867752075, -1.5323270559310913, -1.6104705333709717, 1.1146512031555176, -0.2403213381767273, 0.3089584708213806, -1.2729424238204956, 1.6288316249847412, 0.9729485511779785, -1.399179220199585, -1.405641794204712, -0.7583742141723633, -1.0964411497116089, 0.7305396795272827, -0.2543303072452545, -0.5553657412528992, 0.35924118757247925, -0.2059711068868637, 1.6802066564559937, 0.29370298981666565, -1.171485424041748, -1.3152275085449219, 1.4723399877548218, -0.0006655167671851814, -1.4728748798370361, 0.048826880753040314, -1.941772222518921, -0.7116553783416748, 1.4311233758926392, 0.20386911928653717, -1.2598130702972412, 0.11492238193750381, -1.1548757553100586, -0.04920065030455589, 0.038254544138908386, 0.6550358533859253, 0.8857755661010742, -1.680516242980957, -0.2234760820865631, 0.7698577046394348, 0.0061018499545753, -0.3947787880897522, 0.9215389490127563, 1.0447345972061157, -0.17372113466262817, -0.1504661589860916, 1.9136204719543457, -1.4614622592926025, 0.9542070031166077, 0.8796533346176147, 0.9600415825843811, -0.4083768427371979, 0.0202261283993721, -0.7441156506538391, 2.105384349822998, -0.8965306878089905, 1.8663647174835205, 0.41779568791389465, -0.9471356272697449, 0.1084754690527916, -0.6561342477798462, -0.07620783895254135, 1.1609898805618286], [-0.2218446433544159, 0.41288331151008606, 0.613877534866333, 0.9419043064117432, -0.6675015687942505, -1.4533164501190186, 0.014911382459104061, -1.7325996160507202, -0.9407525062561035, 0.993804931640625, -1.5708168745040894, -0.5328064560890198, 0.10546454042196274, 2.0816116333007812, 0.2290005385875702, 1.6053898334503174, -0.571563720703125, -0.6839119791984558, 0.2729320526123047, -0.6598882675170898, 1.5264310836791992, -1.4165316820144653, 0.43862438201904297, -1.2476710081100464, -0.9300981760025024, -1.3892912864685059, -1.856392741203308, 0.25508397817611694, 0.4731276333332062, 1.9646291732788086, -0.39282020926475525, -1.3115969896316528, 0.5435109734535217, 1.0376694202423096, -0.8880549073219299, -0.40347835421562195, 0.998775064945221, 0.06729438900947571, 0.034681662917137146, 0.5146268606185913, 0.14755871891975403, 0.8917171955108643, -0.7587729692459106, 2.3110382556915283, 1.2972217798233032, -1.9756183624267578, 0.46113520860671997, 0.2741577625274658, 0.5956459045410156, 0.7718747854232788, 1.1711524724960327, 0.1460457146167755, 1.8022980690002441, 0.032598573714494705, -1.4616063833236694, 2.0247745513916016, -0.5499706268310547, -0.7909201383590698, -0.36847877502441406, 0.07467032968997955, -0.8133097290992737, -0.23369263112545013, -0.7550239562988281, -0.8509591817855835, 1.9443732500076294, 1.7051324844360352, -2.0072572231292725, -0.29013362526893616, 0.4822498857975006, 1.1803064346313477, -0.6988587379455566, -0.4940236210823059, -0.09207472205162048, 0.2065260261297226, 2.95109486579895, -0.17992758750915527, 1.079593539237976, 1.486094355583191, 0.6078359484672546, 1.0755071640014648, -0.16525860130786896, -1.3239837884902954, -0.0511368028819561, -2.0862390995025635, 0.16967539489269257, -0.027758438140153885, -0.047093238681554794, -0.5360138416290283, -0.013598345220088959, 1.2596591711044312, 1.7769793272018433, -1.979022741317749, 1.9412477016448975, 0.8421977758407593, 0.23346711695194244, 0.6733512878417969, 0.8052383065223694, 1.6767271757125854, 0.6455028057098389, -0.18804562091827393, 0.8739592432975769, -0.21662305295467377, 0.37251612544059753, -0.13263945281505585, -1.445876121520996, -0.7942376136779785, -0.9617258906364441, 0.9220829010009766, -2.6109957695007324, 0.20382380485534668, -0.5286688804626465, -2.419308662414551, 1.0862029790878296, -0.6256822943687439, -0.8049728274345398, -0.9731770157814026, -0.9749762415885925, -0.6335546970367432, 0.07054990530014038, -0.21317580342292786, 0.14986790716648102, -0.7659560441970825, -0.9510533213615417, 0.775685727596283, 0.2074446827173233, -0.29372090101242065, -0.06281336396932602, -0.12761054933071136], [1.6670403480529785, -0.1768956184387207, 1.725670337677002, -1.5167564153671265, -0.44003134965896606, 0.048213351517915726, -0.6707023978233337, 0.32958462834358215, -1.7366615533828735, 0.978859543800354, 0.050808511674404144, -1.9482760429382324, 0.1281833052635193, 0.42230331897735596, 2.1908271312713623, -0.18618904054164886, -0.69855797290802, -1.1599472761154175, 0.06295858323574066, -1.1146694421768188, -1.1023796796798706, 0.9049901962280273, 0.005969286896288395, 0.10861591249704361, -0.05701328441500664, 0.9232515096664429, 1.339913249015808, 0.8346011638641357, 0.3298114240169525, 0.15335795283317566, -0.747366726398468, 0.7464591860771179, -0.36792197823524475, 0.10693079233169556, -0.051490604877471924, 0.01144234649837017, -1.4149525165557861, -0.5759518146514893, 0.9410862922668457, 0.7344083786010742, 0.1981431543827057, -0.12697559595108032, 0.6798157095909119, 0.4340590834617615, -0.6904511451721191, -2.9098052978515625, 0.12912185490131378, 1.6940351724624634, -1.9078491926193237, 0.9383373856544495, 1.7028777599334717, 1.4697457551956177, -0.5559369921684265, 0.3359204828739166, -0.01316993497312069, 0.5677217841148376, -1.3028937578201294, 1.644720435142517, 1.5100982189178467, 1.07806396484375, -0.2373928725719452, 0.29384806752204895, -0.9278531670570374, 0.33614060282707214, 0.9654854536056519, -0.08146446943283081, -0.2596766948699951, -0.07739735394716263, -1.915992259979248, 1.52409029006958, -0.04335501790046692, 0.035216886550188065, -1.4507156610488892, 2.423572540283203, -0.4968639016151428, 1.0835014581680298, 0.6099899411201477, 0.5440248847007751, 0.584617555141449, 1.410194993019104, 0.14619670808315277, 0.5245010852813721, -0.301608145236969, 0.2321649193763733, 0.3479405343532562, 1.3169822692871094, 1.438963770866394, 0.7296742796897888, 0.4194190800189972, 1.0129005908966064, -0.4138842523097992, -0.9423294067382812, -0.13499420881271362, -1.4214485883712769, -0.02772088348865509, 0.501078724861145, 2.1032016277313232, 0.9345081448554993, 0.564211905002594, -0.3252430856227875, -0.9921952486038208, 0.4311278760433197, 0.12278297543525696, 1.6900031566619873, 0.9373790621757507, -2.922879695892334, -1.4209967851638794, 0.7438976764678955, 0.681761622428894, 1.8680760860443115, -0.25921714305877686, -1.5897105932235718, 0.5444952249526978, -0.44105273485183716, -0.18934577703475952, -0.6434972286224365, 0.36697426438331604, 0.032747358083724976, -0.4047519266605377, 0.31306925415992737, -0.2839014232158661, 1.0186847448349, -0.19765715301036835, -1.4472757577896118, -0.23853087425231934, 0.6496047377586365, -1.0224577188491821, -0.35543742775917053], [-0.006817151326686144, 0.8706572651863098, -0.37177684903144836, -0.3544550836086273, 0.792864203453064, -0.9202584624290466, -0.29332906007766724, 0.8406191468238831, -0.05168132111430168, -0.7524322271347046, 1.4684672355651855, 0.26414141058921814, 0.9266420006752014, 0.04698558524250984, -1.3217968940734863, -1.2830545902252197, 0.9869229793548584, -0.18225981295108795, -0.6126751899719238, -2.105632781982422, -0.9078271985054016, -0.31206071376800537, 1.1348577737808228, 1.0888011455535889, 1.0833309888839722, 0.0019621376413851976, -0.9695906639099121, 0.07706847786903381, -0.8954877853393555, -1.1881256103515625, 1.7096972465515137, -0.36548393964767456, -0.24122220277786255, -0.03241785988211632, 0.09404121339321136, -1.114895224571228, 0.04039841517806053, -0.3560720682144165, 0.19190646708011627, -0.11561308056116104, -1.3943705558776855, 0.10216250270605087, 1.4647542238235474, -0.2774444818496704, -1.072119951248169, 0.010656925849616528, -1.8704509735107422, 0.34640705585479736, 0.0572049543261528, 0.4525834321975708, 0.9531434178352356, 0.17841875553131104, 0.660263180732727, -0.6664554476737976, -1.1577188968658447, -0.8569221496582031, 1.7951469421386719, -2.0175888538360596, 0.20597706735134125, 1.1802263259887695, -0.038786381483078, -1.3295342922210693, -1.1805636882781982, 0.7449964880943298, 0.22648969292640686, 0.6030793786048889, -0.6097357273101807, -1.2404578924179077, 2.267033338546753, -1.8447386026382446, 0.6275142431259155, -0.6921796798706055, -0.8371530175209045, 0.18663708865642548, -2.329401969909668, 0.5419010519981384, -0.26112625002861023, 1.248471736907959, -1.1578006744384766, -0.3708428740501404, 0.6032923460006714, -0.18338394165039062, 1.2167668342590332, 0.15213890373706818, -2.74068546295166, 1.5562660694122314, 0.2734597623348236, -0.5888414978981018, 0.03663342073559761, -0.9418658018112183, -0.12384118884801865, -0.9132907390594482, 0.31971192359924316, -0.34996357560157776, 2.2044177055358887, -1.5786855220794678, -0.9549612402915955, 0.6483140587806702, -0.2188820242881775, -1.3388994932174683, -1.6280394792556763, 1.2600419521331787, -0.28681209683418274, -1.035074234008789, 0.033790722489356995, 0.6372655034065247, -1.8257114887237549, -0.6680780649185181, -0.5618223547935486, 1.48576021194458, -0.3393840193748474, 0.7229838371276855, 0.3199632465839386, 0.16483183205127716, -0.5481902360916138, 0.10160905122756958, 1.7844452857971191, -0.6685019731521606, -0.12115449458360672, 0.5087855458259583, -0.4244735538959503, -1.4698344469070435, -1.7839618921279907, -0.370900422334671, 1.7429580688476562, -0.32227301597595215, 0.855880856513977, -0.8285366296768188], [-0.7258346676826477, 0.6503542065620422, -1.1291064023971558, 0.5042828321456909, -1.728350043296814, -2.5117764472961426, -0.08450772613286972, 0.6054664850234985, 1.3921856880187988, 0.3729904890060425, 0.9109609127044678, 0.6931026577949524, 1.5785027742385864, -1.8831239938735962, 0.8246064782142639, -0.6322450637817383, 1.3085224628448486, 0.08957745134830475, -2.140742778778076, 0.30535778403282166, 0.2915051579475403, -0.19063779711723328, 1.502825379371643, 3.1082518100738525, 0.41951730847358704, -1.2568621635437012, -0.6412025094032288, 1.0541555881500244, -0.6108108162879944, 1.5000561475753784, -0.2084624320268631, -0.6059306263923645, -0.7370836734771729, -0.6130867004394531, 0.8136938214302063, -0.5603231191635132, -2.5801637172698975, 0.2692163288593292, -1.1320785284042358, -1.4437838792800903, -1.3316699266433716, -0.24320584535598755, 0.020125219598412514, 1.31147038936615, 1.0033525228500366, -0.028189877048134804, -1.03609037399292, 1.5828925371170044, -1.0016770362854004, -0.18849416077136993, -0.8130632638931274, -0.43895527720451355, -1.4109704494476318, -0.9390808939933777, 2.296294927597046, 0.6091393828392029, 0.16247870028018951, -0.6817026734352112, -0.8560538291931152, -0.8295933604240417, -0.44562044739723206, 0.10867951065301895, 0.1222921833395958, -0.4076969623565674, 1.0222132205963135, 0.9814363121986389, 0.665877103805542, 1.3471059799194336, 1.0144861936569214, -0.8578988313674927, -1.515700101852417, -0.2703171670436859, -0.18242597579956055, 0.3164568543434143, -0.95777428150177, 2.898493528366089, -0.3782416582107544, -0.6386073231697083, 0.7680052518844604, 1.7969049215316772, -0.8614674210548401, -0.4605727195739746, 0.43978041410446167, 1.604691505432129, 1.6389930248260498, 1.213295817375183, 0.5965196490287781, -0.17588026821613312, -0.3358912169933319, -0.2397783398628235, -0.7426292300224304, 0.8212267160415649, 0.676091730594635, -0.9811277985572815, -0.1124112457036972, 0.5763735175132751, -0.36963582038879395, -0.212765634059906, 0.33617153763771057, -1.2930705547332764, -1.0421741008758545, 0.1853434294462204, -2.923377513885498, 0.1614043265581131, -0.4007115066051483, 0.777037501335144, -0.07326064258813858, 1.2743533849716187, -0.22958879172801971, -0.21278803050518036, 0.7270581126213074, -1.0044176578521729, 0.1377888321876526, -2.3172404766082764, -1.38308584690094, -0.4159771502017975, -0.389163076877594, 2.099414825439453, -2.668869733810425, 1.5583713054656982, 0.4277917444705963, 0.45841267704963684, 0.5638061165809631, 1.1178672313690186, 0.8675143122673035, -0.4348936676979065, -0.9670417308807373, -1.0257960557937622], [0.37197479605674744, -2.093449592590332, 2.207517385482788, -1.040743350982666, -0.9474741220474243, 0.8485539555549622, 0.353816956281662, -0.2982100546360016, -0.7253183126449585, 1.4487226009368896, -0.8348000049591064, 0.04134898632764816, 0.4252181649208069, -0.6585172414779663, 1.5554007291793823, 1.5529534816741943, -0.6225181221961975, -0.8524770140647888, -0.7415375709533691, -0.36850762367248535, -2.048504590988159, 1.7314512729644775, 0.062084004282951355, -0.6387932896614075, -0.28693604469299316, -0.3737780451774597, -0.5468543767929077, -1.6127010583877563, -3.4100077152252197, 1.477806806564331, 0.7254151105880737, 0.19878247380256653, -2.2793967723846436, -0.8749262690544128, -1.332904577255249, -1.2614593505859375, 0.604945957660675, 0.32444649934768677, 0.1520593762397766, 0.06262468546628952, 0.496349573135376, -1.1159945726394653, 0.7325518131256104, 0.5833197236061096, -0.6640627980232239, 2.7965095043182373, 0.7536365985870361, -0.521671712398529, -0.5052353739738464, 1.634560465812683, 1.5945780277252197, 0.5166317820549011, 1.4763519763946533, -1.002570390701294, 1.9229646921157837, -0.6931776404380798, -0.6040016412734985, 1.0820032358169556, -0.31905028223991394, 0.256123423576355, -0.3267075717449188, 1.2949572801589966, -1.1598148345947266, 0.3313535153865814, -0.13453075289726257, 1.246432900428772, 0.5725931525230408, -0.05471525713801384, 2.1238627433776855, -0.23037633299827576, 0.541146457195282, -2.2684481143951416, -0.8469464778900146, 0.305416077375412, 0.23677238821983337, -0.010323084890842438, 0.08930502831935883, 0.025715243071317673, 1.2059341669082642, 0.2703794538974762, 0.4706503748893738, 0.6604426503181458, -0.1795744001865387, -0.7075977921485901, -0.048669926822185516, -0.4656006693840027, -0.9617737531661987, -0.08498990535736084, 0.3289954960346222, -1.4867403507232666, 0.2595749497413635, -0.9144803881645203, -0.34002113342285156, 0.7532406449317932, 1.1024836301803589, 0.9825917482376099, 0.02256365306675434, -1.418349266052246, -0.47914308309555054, 1.2342250347137451, 0.5293542146682739, -0.6263045072555542, 1.6048481464385986, 0.41892457008361816, 3.3763909339904785, -0.5577920079231262, -0.06769789010286331, -0.9239026308059692, 0.39159145951271057, -0.7558256983757019, 0.46375080943107605, 0.7972218990325928, 0.45386263728141785, -0.9137935638427734, -0.6017060875892639, 0.19502834975719452, 0.6648034453392029, -0.7614090442657471, -0.23938210308551788, 1.6023929119110107, -0.6323022842407227, 0.5439485907554626, 0.3023745119571686, -0.282604843378067, 0.06087160483002663, -0.1803155541419983, -0.09619323164224625, -0.3590713441371918], [-1.90224289894104, -0.158871129155159, 0.3333779275417328, 0.02121693082153797, -2.512896776199341, -0.23192289471626282, -1.1685552597045898, 1.3103569746017456, -1.0653860569000244, -0.5476825833320618, 0.9706496596336365, 0.7100138068199158, -0.018021762371063232, 0.3547441363334656, -1.0885615348815918, 0.4170688986778259, 1.807108998298645, -0.22540068626403809, 0.4619079828262329, -0.2936898469924927, -1.513810157775879, 0.0598687008023262, 0.6517658233642578, -1.1491467952728271, 0.9519758820533752, -0.01164940930902958, -0.40305042266845703, -1.8479565382003784, -1.2652065753936768, 1.128728985786438, -1.2436842918395996, -0.27843090891838074, -0.07324931770563126, 0.4575690031051636, 0.9165898561477661, 1.5311036109924316, 2.2067182064056396, -0.7677581310272217, -1.4316296577453613, 1.9177284240722656, 0.5772014260292053, 1.314812183380127, -1.4316290616989136, 1.2212811708450317, -0.1826387643814087, 1.5130929946899414, -1.1924885511398315, -0.1607782244682312, 1.2097676992416382, -0.48406746983528137, -0.5433106422424316, -0.2031116634607315, -0.798462450504303, -1.9812288284301758, -0.09655655920505524, 0.4950880706310272, -1.9533655643463135, -1.1040409803390503, 1.5004185438156128, 1.156367301940918, 0.3464082181453705, -1.46290922164917, -1.429982304573059, 0.6107665300369263, -0.4049195945262909, -0.6638866066932678, -0.17054758965969086, 1.247018575668335, 0.9137541055679321, 0.8595224022865295, -0.2432529777288437, 0.018930526450276375, 0.25801336765289307, -0.09152347594499588, 0.8237398266792297, 0.5147963762283325, -0.7959043383598328, -0.9873747229576111, 0.5289035439491272, -0.6136803030967712, -0.3091985881328583, -0.21401305496692657, 0.62619948387146, 0.8559893369674683, -0.5514700412750244, -0.5025750994682312, 0.6896435618400574, -0.5070998668670654, -0.2817957401275635, 0.3606778681278229, 0.8667028546333313, 1.0586802959442139, -0.154384046792984, -0.4733697772026062, -1.0383564233779907, -0.9786238670349121, -2.0611884593963623, 0.7400951385498047, 0.2059272676706314, 0.1938478797674179, -0.5356141924858093, -0.25497376918792725, -0.7834951877593994, -0.5744054317474365, -2.5222666263580322, -0.8534173965454102, 0.07611039280891418, 0.3407120406627655, 2.3937859535217285, -0.8061663508415222, -0.33947548270225525, -0.041030559688806534, 0.5070490837097168, 0.1073099747300148, -0.09163623303174973, -1.9471441507339478, -0.96329665184021, 0.6397266387939453, 1.4699770212173462, 1.4486654996871948, -0.17374607920646667, 0.2938459515571594, -1.7652831077575684, 2.008971929550171, -1.794545292854309, 0.6826202869415283, 0.5198675990104675, 0.5496917963027954], [0.7242059707641602, 1.0664674043655396, -0.290727436542511, -0.327944815158844, 0.15730158984661102, -1.6774532794952393, -1.2758909463882446, -1.7725327014923096, -0.25686609745025635, -0.1246669813990593, 0.6164315342903137, 0.9507557153701782, -0.4944698214530945, -0.6617539525032043, -0.9633676409721375, -0.6658565402030945, 0.5015702247619629, 0.23295943439006805, -0.2292712777853012, 1.171357274055481, -1.2862341403961182, -1.162933111190796, -0.11489561200141907, 0.3049720823764801, 2.0498363971710205, -0.8768438696861267, -2.184422492980957, -0.3680953085422516, 0.32223090529441833, 0.07431994378566742, -1.0573526620864868, 1.2892496585845947, -0.7119978070259094, 1.7313339710235596, 0.28745827078819275, -1.4176489114761353, 1.3855558633804321, -0.3739689290523529, -0.16779738664627075, 0.7103153467178345, -1.7477120161056519, 0.43338558077812195, -0.9619821310043335, 0.5501086115837097, 0.5129369497299194, -0.30366250872612, -0.06046091020107269, -0.976003110408783, -1.382534146308899, 0.5125504732131958, 0.23971213400363922, 1.8768203258514404, 2.5692851543426514, -0.41036462783813477, -0.6577705144882202, 0.3747493028640747, 0.22880129516124725, -1.6163662672042847, 1.1730568408966064, 0.3856845796108246, -0.7270981073379517, 0.33624228835105896, 0.37471136450767517, 0.09096021950244904, -0.7988207340240479, -0.12746168673038483, -0.07829400897026062, -1.0659457445144653, -0.07102145999670029, -0.7654603719711304, -0.27471694350242615, 0.41457468271255493, 0.14080365002155304, 0.48445677757263184, -0.810109555721283, 0.2890060842037201, 0.7018107175827026, -0.3388465642929077, -0.820682168006897, 0.46082285046577454, -0.534784734249115, -2.094252586364746, -2.273460865020752, 1.1575020551681519, -0.32230299711227417, 0.70100998878479, 1.3851252794265747, 1.0308606624603271, -1.6027591228485107, -0.8259715437889099, 1.1695547103881836, 0.2427307814359665, 0.21760235726833344, -0.34668633341789246, -1.409441351890564, 0.3887992799282074, -1.9820562601089478, -1.3865628242492676, -0.4614228308200836, 0.2304258644580841, 1.3969693183898926, -0.10439911484718323, 1.2017261981964111, 0.45510637760162354, 0.019859574735164642, -1.0220370292663574, 0.07914157211780548, -0.24909579753875732, 1.2440606355667114, -0.6404922008514404, 0.5702099204063416, 0.012305361218750477, 0.4915119409561157, -0.6098846793174744, -1.0420973300933838, -1.2685190439224243, -1.4510469436645508, 1.203069806098938, 1.278320550918579, -0.5930480360984802, 1.5088422298431396, -0.6551601886749268, -0.6544999480247498, 0.5572515726089478, -1.5824434757232666, 0.626147985458374, -0.2168692797422409, -0.38498127460479736], [-0.28012794256210327, -0.8711268901824951, -0.6289159655570984, 2.621203660964966, -1.0112298727035522, 1.7113083600997925, 0.7631597518920898, -0.17752403020858765, -0.7169491052627563, -0.08914820849895477, -0.8330756425857544, -0.15214534103870392, -0.233809694647789, -1.712600827217102, -0.2413458675146103, 0.34060391783714294, -1.0503705739974976, -0.31738924980163574, -0.8288105130195618, -0.7138050198554993, 0.17779883742332458, 0.13412980735301971, -0.21206893026828766, 0.7020806670188904, 0.32442760467529297, 1.5668041706085205, -1.0498956441879272, 0.28056225180625916, 1.3377423286437988, 0.39101341366767883, 0.3272267282009125, 0.4407637417316437, -1.497257113456726, -0.5394710898399353, -0.20793525874614716, 1.2022899389266968, -0.14635825157165527, 0.3873312175273895, 0.2505775988101959, 1.547475814819336, 0.7240371108055115, -0.07216444611549377, 0.7447760105133057, 0.7533393502235413, -0.7019867300987244, 0.8725572228431702, -1.0539097785949707, 1.5318653583526611, 0.16424019634723663, -0.4595496654510498, 1.145179271697998, -0.33498427271842957, 0.3077842593193054, 0.7339674830436707, -1.34089195728302, -0.8880772590637207, 0.8624143004417419, 0.46525025367736816, 0.9293149709701538, 0.03752619028091431, 1.4381295442581177, 0.9170344471931458, 0.5356391668319702, 0.22801364958286285, 0.1038832888007164, -0.058615542948246, 0.48040586709976196, -0.24935266375541687, -0.6905471682548523, 1.1610491275787354, 2.2542567253112793, 0.8386175632476807, 0.6288226842880249, 0.5600385069847107, 1.0301473140716553, 0.6219842433929443, 0.6500443816184998, -2.439257860183716, 0.837981104850769, -0.6396063566207886, 0.35380664467811584, 0.6771210432052612, 0.6273419260978699, 0.3825659453868866, 1.1480234861373901, -0.024742646142840385, 0.40474754571914673, -0.82086181640625, -0.07794907689094543, 0.47488895058631897, -1.1006247997283936, -1.018669605255127, 1.757949948310852, 1.579319715499878, -0.9633601903915405, -0.415301114320755, 0.29113924503326416, -0.6230173110961914, -0.0866864025592804, 0.0776248574256897, -0.7770290970802307, 0.7234606742858887, 0.4202669858932495, 0.22055061161518097, 0.2591205835342407, -2.7950947284698486, 2.245898485183716, 1.1576473712921143, -0.6324850916862488, 0.1966911107301712, -1.2768621444702148, -1.0942504405975342, 0.2348089963197708, 0.9256511330604553, -0.2715585231781006, 0.037284303456544876, -0.9858050346374512, 0.7910052537918091, -0.22468797862529755, 1.1040695905685425, -0.35838615894317627, 1.5362694263458252, -0.65435391664505, 1.148768663406372, 0.874259889125824, 1.6122405529022217, -2.044886589050293, 0.9561627507209778], [-0.45741957426071167, -0.7902386784553528, -0.2378256618976593, -1.15081787109375, 0.38749921321868896, 0.061511535197496414, 1.1288288831710815, -0.23878665268421173, 0.6361862421035767, 0.12902414798736572, 0.43024858832359314, -1.1338914632797241, -0.529725193977356, 1.2826355695724487, -1.5631107091903687, -0.5676382184028625, -0.17053848505020142, -0.4540855884552002, -2.6002116203308105, 0.5709611177444458, -0.24078452587127686, -0.68528151512146, 2.5811147689819336, 0.10445863753557205, -0.6065776348114014, -0.31954237818717957, -0.6810837388038635, 0.6463186740875244, -1.1478270292282104, 3.1819894313812256, 0.6529814004898071, 1.0536649227142334, 0.8375600576400757, 1.6746008396148682, 0.10692796111106873, -1.144087314605713, -1.189146876335144, -1.7538145780563354, -0.20804272592067719, 2.162240743637085, 0.9147980809211731, -0.4370826482772827, -2.5846405029296875, -2.196216583251953, -0.54925537109375, 0.6605151891708374, -0.19399778544902802, -1.3934904336929321, 0.14226660132408142, 0.22585371136665344, 1.6717588901519775, 3.0736773014068604, -0.6617487668991089, 0.8801957368850708, -0.9145210385322571, 0.2758447825908661, -0.47378358244895935, -0.3503457009792328, 1.2503116130828857, -1.0730233192443848, -1.8652368783950806, -0.33554521203041077, -2.0697174072265625, -1.5116338729858398, 1.5527596473693848, -0.15479132533073425, -0.5430982112884521, -1.8702656030654907, -0.31545138359069824, -0.4360906779766083, -0.7633314728736877, -0.0019396285060793161, -0.31364938616752625, 0.3295121192932129, 0.23890705406665802, 1.252701997756958, -0.6886693835258484, -0.8014563918113708, -0.8398765921592712, -0.14838822185993195, -0.5509068965911865, 0.5987340211868286, 1.5632067918777466, 0.962192177772522, -1.1160697937011719, -0.12471991777420044, -0.6525467038154602, -0.8731191158294678, 0.1470944732427597, 0.5747941732406616, 0.8663025498390198, 0.21514058113098145, -0.8245996236801147, 0.0961613655090332, 0.44766032695770264, -0.9810804128646851, -0.8388566374778748, 0.26018962264060974, 0.5475080609321594, -1.464795708656311, -0.6595470905303955, -0.930952250957489, 0.772483229637146, 1.2146811485290527, 1.0916264057159424, 0.21432870626449585, -1.6719841957092285, 1.4852030277252197, -0.8099521994590759, -1.9352673292160034, -0.9333491325378418, 1.4028635025024414, 1.0789059400558472, 1.4680545330047607, -0.4357515275478363, -1.5880314111709595, 0.2842789888381958, 0.9866324663162231, 0.27368032932281494, -0.010130473412573338, 0.3701798915863037, -1.281138300895691, -0.5111348628997803, -0.35717687010765076, 0.14898933470249176, 1.262971043586731, -0.04417720064520836, 2.019275426864624], [-0.926596999168396, -0.422541081905365, 0.8650330305099487, -0.5716698169708252, -0.3640124499797821, 0.3447551727294922, -0.8527476787567139, 0.4148639440536499, -0.36612746119499207, -1.0744613409042358, -0.6921862959861755, 0.03432726114988327, -0.8112389445304871, -0.08419937640428543, 1.5286495685577393, 0.08043015003204346, 1.417167067527771, 0.4503672122955322, 0.8903393745422363, 0.8484010696411133, 0.869613766670227, 0.7636463642120361, -0.6169378757476807, -1.3404700756072998, -0.0033359061926603317, -1.3519365787506104, 0.6415392160415649, -0.1663099080324173, -0.3134535551071167, -0.38847383856773376, 0.28336912393569946, 0.9100525379180908, 0.9204249382019043, 0.15440179407596588, -0.22439852356910706, 0.1703348606824875, -0.9967156052589417, -0.6139271855354309, -0.022460319101810455, 0.8563187122344971, 0.2139279842376709, -0.156615749001503, -0.1956038773059845, -1.2899551391601562, -0.03502650558948517, 0.37606048583984375, -0.32806655764579773, 0.29534026980400085, -0.10021156072616577, 0.7674620747566223, -0.6544336676597595, 0.0301953237503767, -0.08927889168262482, -0.535365879535675, -2.158698558807373, 0.6260600090026855, -0.2135995328426361, -0.7042791843414307, -0.2450701892375946, 2.824496269226074, 0.11174163222312927, -0.7898096442222595, 1.632393717765808, -0.23402781784534454, -2.7927608489990234, -0.3521234691143036, -1.1502251625061035, 0.21206699311733246, 0.6033467650413513, 1.1958889961242676, -0.9612026214599609, -1.1775723695755005, 0.11649208515882492, 0.6931413412094116, -0.2048463225364685, 0.0004274844832252711, -1.0954551696777344, -0.8076611757278442, -0.2794082760810852, 0.936234176158905, -0.15840311348438263, 0.15810488164424896, 0.9376841187477112, -0.9014680981636047, 0.765900731086731, -0.2602880299091339, 2.0731849670410156, 0.6740325689315796, 2.0914628505706787, 0.8550918698310852, 0.16324187815189362, 0.6848181486129761, -0.633004903793335, 1.0426414012908936, 0.008296117186546326, 0.05242563411593437, 0.15293312072753906, 0.09853308647871017, -0.45394638180732727, 0.8100224733352661, -1.6291465759277344, 0.24075260758399963, 0.35624486207962036, 0.613297164440155, 0.8436447381973267, 0.09192803502082825, -0.98762047290802, -0.16879414021968842, -0.0422838069498539, -1.6389232873916626, 0.12571978569030762, -2.610013246536255, 1.6495585441589355, -0.1433030068874359, 0.8111469745635986, 0.24831590056419373, 1.3126721382141113, -1.3811582326889038, -0.3627096712589264, 0.4115290641784668, -0.8751124739646912, 1.5411802530288696, -1.5256049633026123, -0.5538002848625183, -1.3961604833602905, -1.4265198707580566, -1.09523606300354, -0.29033949971199036], [1.9975128173828125, -0.056826137006282806, 0.27947044372558594, 1.23307204246521, 0.6681020855903625, 2.0597379207611084, 2.980778455734253, -1.0066028833389282, -0.2046925574541092, -0.14229221642017365, 1.1832048892974854, -0.8798092007637024, -0.8140751123428345, 0.003171368967741728, -0.20675252377986908, 1.4072362184524536, 1.2444361448287964, -0.3212948143482208, -0.03012828156352043, 0.07113399356603622, -2.329411506652832, -1.7598137855529785, 0.7204994559288025, -2.4344263076782227, -1.2711890935897827, -1.0366970300674438, -0.041701458394527435, 0.28270477056503296, -0.4009542465209961, 0.6574774384498596, 0.6197068691253662, -1.394580364227295, -1.2193481922149658, -0.32195448875427246, -0.5322324633598328, -0.051465123891830444, -0.252730131149292, -0.5075339078903198, 0.1498185396194458, -0.34999197721481323, 0.8434576988220215, 0.5166369676589966, -1.7555996179580688, 0.2471274733543396, 1.485539197921753, 0.4473431706428528, -2.287020444869995, 1.337153434753418, -1.1046407222747803, 1.8301939964294434, -0.7376194596290588, -0.7891669869422913, 0.12553724646568298, 1.21176278591156, 0.2803889214992523, -0.6005274057388306, 0.34575632214546204, 0.797712504863739, -1.2729476690292358, -0.89711993932724, 0.2378406524658203, 0.2582409381866455, 1.8023868799209595, 0.13755592703819275, -0.24836014211177826, 0.11085651814937592, 0.191806823015213, 0.012818931601941586, -0.29942670464515686, 0.7152671813964844, -1.317432165145874, 0.3544274568557739, 1.5219776630401611, 0.6242486238479614, -0.19251269102096558, 0.39382433891296387, -0.9546547532081604, -0.24014118313789368, -0.19074194133281708, -1.5585224628448486, 0.671149730682373, -0.7834279537200928, 0.4097065031528473, -0.7066460847854614, 1.378553867340088, 0.13849085569381714, -0.11433272063732147, 0.2040991187095642, -0.6140810251235962, -0.7504740357398987, 1.341715931892395, -0.22954386472702026, -1.0071978569030762, 1.6039981842041016, -0.18397772312164307, -0.6877329349517822, -0.12059567123651505, 0.18145349621772766, 0.4371251165866852, -2.0247886180877686, 1.6971498727798462, -1.2937690019607544, -0.8197034597396851, 0.31713154911994934, 0.765323281288147, 0.5039919018745422, -0.04117438569664955, -0.44356033205986023, 0.5783794522285461, -0.29941290616989136, -0.8358032703399658, 1.3349767923355103, -0.3663065433502197, 0.3364136219024658, 0.5086880922317505, 0.8469926714897156, -0.40605902671813965, 0.9739205837249756, 0.2622363269329071, 1.512715458869934, 1.2222133874893188, 0.9723448753356934, 0.7572200894355774, -2.6597466468811035, -0.2935994565486908, -0.5297064781188965, 0.1805686056613922, 1.8223243951797485], [-1.9565192461013794, -0.02187226712703705, 0.5526830554008484, -0.16898910701274872, 0.15040116012096405, -1.1609586477279663, -0.6334825158119202, -1.0329978466033936, -0.6410517692565918, 0.7230572700500488, 0.0711149275302887, 0.5507219433784485, -0.09602192789316177, 1.3682398796081543, 0.5909014344215393, 0.051138147711753845, 0.33702021837234497, 1.1876217126846313, -0.5512771010398865, -0.14447003602981567, -0.46227094531059265, -0.2606242001056671, -0.2938423156738281, 0.8291589617729187, -1.304474949836731, 0.8939625024795532, -0.4724578857421875, -1.394156575202942, -1.1414527893066406, 0.09920133650302887, -0.004567279946058989, 1.3743137121200562, -1.156123161315918, -0.5809243321418762, -1.2066112756729126, -1.2711905241012573, -0.061181243509054184, 0.6517888903617859, -0.800593376159668, 1.2914241552352905, 0.3056289851665497, 0.18309380114078522, 2.8519513607025146, -0.6685591340065002, 0.9146896600723267, 0.03029804117977619, -0.37284478545188904, 1.1561704874038696, 1.199155569076538, 0.45201534032821655, 2.5796658992767334, 0.3378121256828308, 0.1687568724155426, 0.6088156700134277, 0.6052511930465698, 0.5717740058898926, -0.6034125089645386, -0.1176404282450676, 0.6712243556976318, 0.7692337036132812, 0.6162450909614563, -1.3271880149841309, 0.6738114356994629, 0.6889819502830505, -0.19229397177696228, 0.04249711334705353, 0.83053058385849, -1.6353965997695923, 2.159803867340088, 0.43075692653656006, -0.08955435454845428, -0.04712125286459923, 0.6707060933113098, -0.05147331580519676, -1.108959436416626, 0.39533764123916626, -0.3167250454425812, -0.4353407919406891, 0.9265892505645752, -0.384378045797348, -2.1957175731658936, 1.5333497524261475, -0.5935874581336975, 0.816700279712677, 1.1333013772964478, -0.09783732891082764, -0.5056440830230713, 0.028426596894860268, -0.5148828029632568, 0.9759223461151123, 0.8649300336837769, 0.1741025298833847, -1.5871468782424927, -0.5599095821380615, 1.4219300746917725, -0.7328760623931885, -1.2328165769577026, -1.6053986549377441, -0.6225056052207947, -1.2302830219268799, 0.2470618039369583, -0.55918949842453, -1.416837215423584, 0.8514336943626404, -0.691081166267395, 1.105254054069519, -0.29214510321617126, 0.25349557399749756, -0.5892441868782043, -1.3460818529129028, 0.29031017422676086, -0.12074894458055496, -0.6002455949783325, -0.7902063131332397, 0.2532210052013397, -1.6234302520751953, -0.5461024641990662, 0.10087791830301285, -0.8085764050483704, 1.1774498224258423, -2.31321120262146, 0.92789226770401, 1.0782586336135864, -0.3928154706954956, 0.7538931369781494, 0.055608637630939484, 0.9519078135490417, -0.908057689666748], [1.1069978475570679, 0.8264669179916382, -0.37049755454063416, -0.6951740980148315, 1.661455512046814, -1.2523548603057861, 1.0684531927108765, 0.46244707703590393, 0.16785690188407898, 0.4849928021430969, 0.7984657883644104, -1.1045492887496948, -1.6070425510406494, 0.24663065373897552, 0.1383877843618393, 0.834835410118103, -0.237540140748024, 1.4258962869644165, -0.5822167992591858, -2.4078550338745117, -0.2622601091861725, -1.925137996673584, -1.0458002090454102, 1.1700754165649414, -0.229569211602211, -0.8287023305892944, 0.6792439818382263, -0.3399292826652527, -1.341437578201294, -0.4163462817668915, -1.2084980010986328, -0.36358392238616943, 1.1793135404586792, -0.12522420287132263, -0.9243224859237671, -1.1377099752426147, -0.39660879969596863, 2.393662214279175, -0.7371578216552734, -0.8163797855377197, 0.26339519023895264, -1.0076714754104614, 0.04351144656538963, -1.8189337253570557, -1.1521379947662354, 1.9650086164474487, -0.3711545467376709, 1.2314555644989014, -0.09731657058000565, -0.5308448076248169, 0.05885232239961624, -1.2455105781555176, 0.12858766317367554, -0.05199853330850601, 1.2969080209732056, 0.234497532248497, 0.4525187015533447, 2.1943612098693848, -0.2430848330259323, -0.13468602299690247, -0.6693718433380127, -0.21963007748126984, 0.2527795135974884, 1.6988073587417603, -1.2049487829208374, -0.7115169763565063, -0.02240181155502796, 0.75433748960495, 0.9715186357498169, 0.8114479780197144, -0.8397619724273682, 0.6722663640975952, 0.01195472851395607, 0.04674932360649109, -0.32816585898399353, 0.3357747793197632, -1.625604271888733, -0.8602596521377563, -0.6436725854873657, -2.072108268737793, 0.6876793503761292, 0.6028656363487244, 1.199894666671753, 0.4862738847732544, 2.20621657371521, -1.340400218963623, 1.008809208869934, -1.863813877105713, 1.717093825340271, -0.22227874398231506, -0.7605892419815063, -0.16026070713996887, 0.16895230114459991, 0.702458918094635, 0.9044440388679504, -1.2808095216751099, -0.5554654002189636, -0.1887960433959961, -0.7910043597221375, 0.9783775210380554, -0.7031047940254211, -1.6854203939437866, 0.0342080257833004, 1.1431357860565186, -0.5081846714019775, -0.5089244246482849, -0.32325661182403564, -1.3633753061294556, -0.6217721104621887, 0.33647799491882324, 0.23946595191955566, 2.448838233947754, -0.06305268406867981, 0.20560579001903534, 0.027380075305700302, -0.8489142656326294, 0.7764206528663635, -0.4782031774520874, 0.7616060972213745, 0.10723871737718582, -1.5276594161987305, 0.3732438385486603, 0.9180463552474976, -1.3541343212127686, 1.4169493913650513, 0.8625967502593994, -0.4538177251815796, 1.0699957609176636], [0.7868616580963135, 0.45749107003211975, -1.2200502157211304, -1.1680498123168945, 0.08776307851076126, -0.5231244564056396, -1.125124454498291, -0.7947250008583069, 0.596686840057373, 0.8505003452301025, -0.4055578112602234, 1.4557039737701416, 0.5825892686843872, 0.011885746382176876, 0.8837677836418152, 1.558396816253662, 0.4953148663043976, -2.0059523582458496, -1.1074739694595337, 0.4225018620491028, -1.5962918996810913, -0.6367225646972656, 0.3698185086250305, -0.5501846075057983, 1.101681113243103, 0.5016711354255676, -0.30736660957336426, 0.9129266738891602, -1.4364405870437622, -0.6057685017585754, 0.4364194869995117, -0.8236686587333679, 0.07249945402145386, 0.8055598735809326, 0.5942201018333435, -1.3700073957443237, 0.1472860723733902, -0.2316327840089798, 0.2585599422454834, 0.006361289415508509, 0.09039923548698425, -0.7257371544837952, 0.5916891098022461, 0.8150728940963745, -0.4908939301967621, -1.5252795219421387, -1.494140386581421, -1.814149260520935, -0.366990864276886, -1.0997979640960693, 0.1824372261762619, -1.4345059394836426, -1.0386106967926025, -0.04102218523621559, -0.6568408608436584, -1.2811455726623535, 0.02664668671786785, -0.6952641010284424, -0.02098161168396473, -0.321800172328949, 1.5861544609069824, 1.7110772132873535, -1.2316259145736694, 1.7285234928131104, 0.5631247162818909, -0.04321535676717758, -1.3448235988616943, 0.36543068289756775, -2.6393768787384033, 2.2115561962127686, -0.7375256419181824, 0.6418821215629578, -0.6171380281448364, 0.3145092725753784, -0.14898037910461426, 0.1825902909040451, 1.3219996690750122, 0.4139239192008972, -1.0920720100402832, -1.496592402458191, -1.4453684091567993, -0.11477493494749069, 0.8163623213768005, 0.6894641518592834, 0.7903963327407837, 0.9518983364105225, 0.9263201355934143, 0.25529786944389343, 0.5204135775566101, 0.13766834139823914, 0.42808645963668823, -0.5421414971351624, -0.01281761284917593, 0.5241172313690186, -0.06068629398941994, 1.0436055660247803, 0.7341929078102112, -0.9693706035614014, -0.8456896543502808, 0.6607226729393005, -0.3885364830493927, 0.0025427655782550573, -0.9773417711257935, 0.1382666528224945, -1.015517234802246, -1.352328896522522, -0.28823286294937134, 1.5347168445587158, 0.9053560495376587, 1.947392225265503, 0.39391565322875977, 0.1448812633752823, -0.29962456226348877, 2.1358630657196045, 1.4996626377105713, 0.46085798740386963, -0.4716319739818573, 1.597914695739746, -1.2240517139434814, 0.36069855093955994, -2.4318532943725586, -0.19004769623279572, 0.7792468667030334, -1.1526607275009155, 1.2703585624694824, 2.1232988834381104, 0.7149300575256348, -0.5228609442710876], [0.15947295725345612, 1.2454826831817627, -0.9876230359077454, -1.174161672592163, 0.7727431058883667, 0.18061880767345428, 0.8114404678344727, -0.8448620438575745, 1.6621726751327515, -0.030809221789240837, -0.9686986207962036, -1.0224450826644897, 1.2909948825836182, -0.08389855921268463, 1.137896180152893, -1.9221131801605225, -0.9627647399902344, 0.9509420394897461, -0.39939913153648376, -0.9345201253890991, -0.6379724144935608, 1.038740634918213, -0.08799134194850922, -0.39771515130996704, -0.3113649785518646, 0.07791934907436371, 1.2929003238677979, -0.743751049041748, -0.9290850758552551, -0.5920833945274353, -1.486378788948059, -0.1137256771326065, 0.4758647382259369, -0.17134620249271393, -0.24816708266735077, 2.1710081100463867, -0.8763564229011536, -2.643846035003662, 0.15317532420158386, 0.09876884520053864, 0.98875892162323, -0.9229230880737305, 0.7829442620277405, -0.3190309405326843, -1.2505074739456177, 0.9480518698692322, -0.16035433113574982, -0.8059355616569519, -1.0596753358840942, -1.2294070720672607, 0.16503660380840302, 0.2429582178592682, 0.05963483452796936, -1.8295649290084839, 1.3189548254013062, -1.3730813264846802, 0.570341169834137, 0.8258615732192993, -2.770380735397339, 0.9596796631813049, 0.9387661218643188, 0.7304481863975525, -0.18855643272399902, 0.27562156319618225, 0.7953096032142639, -0.05103100463747978, 1.0652005672454834, -1.4884949922561646, 0.6798794865608215, -2.348921775817871, 0.8388240337371826, 0.9076080918312073, 0.3072749078273773, 0.002166177611798048, 1.1400549411773682, 0.04183585196733475, 0.7344988584518433, -0.11367201060056686, 0.9163715839385986, -0.36440953612327576, -1.5470046997070312, 1.6013824939727783, -0.8549374341964722, 1.5086514949798584, 0.009722677990794182, 0.8299731612205505, 0.13107480108737946, -0.11281906813383102, 1.319860577583313, 0.013978996314108372, 1.2001157999038696, -0.4315066337585449, -0.7778300642967224, 0.03566187620162964, 0.1975427269935608, 0.5323827862739563, -1.8313488960266113, -0.051373645663261414, 0.40173110365867615, 1.7160090208053589, 0.21719413995742798, 0.42273202538490295, -0.22591359913349152, 2.952516794204712, 0.02251669019460678, 0.2728956937789917, 0.048270583152770996, -1.651281714439392, 0.20531336963176727, -1.3803274631500244, 0.6075617671012878, -2.8234570026397705, 2.037956476211548, -1.00980544090271, -0.3584810197353363, -0.4569615125656128, 0.5312120318412781, 0.9861026406288147, -0.5211532711982727, 1.2177183628082275, 0.10748474299907684, 1.280913233757019, 0.8346617221832275, 0.556205153465271, -0.8578832149505615, 1.4400736093521118, -3.066606283187866, -1.046533226966858], [-0.6258008480072021, -0.23625649511814117, 1.2014695405960083, 0.1528787463903427, -1.6128236055374146, -0.4833351671695709, 1.4277323484420776, 0.8340650796890259, -0.5778266787528992, -0.09559784084558487, 1.2294838428497314, -0.39308440685272217, 0.11564837396144867, -0.4412427544593811, -1.5661494731903076, 1.1542960405349731, -0.8287347555160522, 0.11497589200735092, 0.6509788036346436, -1.0604918003082275, 0.2248496264219284, -0.8897160291671753, 1.3566738367080688, -1.391953706741333, -1.287743330001831, -0.5231711268424988, -0.6818554997444153, -0.405266672372818, 0.4977400302886963, -0.5351660251617432, 0.7798973321914673, -1.4413864612579346, 1.3941614627838135, 0.2788461744785309, -0.658332347869873, -0.9317545890808105, 0.4122232496738434, 0.36907100677490234, 1.9128494262695312, 0.10647273063659668, 1.0389457941055298, 0.009663259610533714, -0.3077285587787628, -1.122637152671814, -0.8606592416763306, 0.02414468489587307, 0.40859225392341614, 0.6892943382263184, 0.2521013021469116, -0.6264330744743347, -0.5007441639900208, 0.7978570461273193, 0.555703341960907, -1.280672311782837, 0.48069527745246887, -0.4264509081840515, 0.08883308619260788, -0.4376184046268463, 1.243764877319336, 0.4600996673107147, 0.512706458568573, 0.2297709733247757, -0.099027119576931, -0.23260119557380676, -0.9718952775001526, 0.24422287940979004, -0.9409699440002441, 0.5453519821166992, -0.03035752847790718, 0.212980717420578, -0.4529520869255066, 1.002704381942749, -0.23147182166576385, 0.6052400469779968, -0.9021997451782227, 1.7871886491775513, 0.2716990113258362, 0.5287309885025024, -0.6016189455986023, 0.3410912752151489, 0.03630445525050163, 0.6800901293754578, 0.7872609496116638, 1.5601460933685303, 2.0953409671783447, -0.3060168921947479, 0.1652214676141739, -0.6902326345443726, -0.6660192012786865, -2.002525568008423, -1.4237645864486694, -0.3757295608520508, -1.7049627304077148, 0.8197900652885437, 0.8697239756584167, -0.421806275844574, -0.2616705000400543, 1.0249838829040527, 1.66177499294281, 0.2462027668952942, -0.07907728850841522, -0.18115639686584473, -1.785416603088379, 0.4156492054462433, -0.6685956120491028, -0.48661503195762634, -0.4697469174861908, -2.2258198261260986, -1.1498934030532837, -0.05233515053987503, 0.13782908022403717, -0.12315675616264343, 1.0665467977523804, 1.2490969896316528, 1.4254359006881714, 0.9317101240158081, 0.7368716597557068, 0.6187567114830017, 0.9387617111206055, -0.4960884153842926, -0.05898779258131981, 0.7630853652954102, 0.4910706579685211, 0.42627257108688354, 0.892035961151123, -2.1289405822753906, 0.40173056721687317, -0.4702025055885315], [0.24034833908081055, -0.5921466946601868, -1.5328410863876343, -0.7853347659111023, -0.2925299406051636, -0.19375094771385193, 0.5377582907676697, 1.1965938806533813, -0.9836156964302063, 0.26212817430496216, -0.4618532061576843, 0.5325379371643066, 0.43883374333381653, 0.59602290391922, -2.222550392150879, 0.27653101086616516, 1.573559284210205, -1.374273657798767, 0.39908793568611145, -0.3008616864681244, -0.42189574241638184, 0.6227395534515381, 1.3593930006027222, 0.7417345643043518, 1.5241978168487549, -0.5965118408203125, 1.3668978214263916, -1.0423169136047363, -0.23177459836006165, 0.8709742426872253, -1.5480947494506836, 0.1097477376461029, 1.2862114906311035, -1.1639149188995361, 0.42868945002555847, -0.09979035705327988, -1.2503719329833984, 2.100687026977539, 0.020138509571552277, 0.3151625394821167, -0.29582130908966064, 0.6853501796722412, 0.7266222834587097, -0.49633651971817017, 0.5754407048225403, -2.149583339691162, -0.9787327647209167, 0.23049092292785645, -0.8634735941886902, -1.5828118324279785, -0.19982559978961945, -0.5805479288101196, 0.48762989044189453, -0.6292324662208557, 1.0705780982971191, -0.6533297300338745, -0.6631169319152832, 1.0497241020202637, -0.12200944125652313, 0.18530842661857605, 0.7847468852996826, 1.3907934427261353, 1.6991512775421143, 1.6467578411102295, 1.4682530164718628, 0.786673903465271, 1.1400632858276367, -0.37729695439338684, 0.6356887221336365, -0.6270962357521057, -0.11336303502321243, -1.0656609535217285, -0.47863009572029114, 0.7045500874519348, 1.3966819047927856, 1.1639422178268433, -1.5243611335754395, 1.007758378982544, -0.8479716777801514, 1.9373334646224976, 0.8604797124862671, -1.8385123014450073, 0.8969115614891052, 0.3125726878643036, -0.2782122790813446, 0.17952273786067963, -0.12345249950885773, -0.06457575410604477, -0.011594394221901894, -0.8553733825683594, -0.6934311985969543, -0.018041687086224556, -0.9900569319725037, 0.36590686440467834, 0.08208017796278, 0.7456742525100708, 0.6070771217346191, -0.36787331104278564, 1.0487251281738281, -0.6529297828674316, -0.43928462266921997, -0.4746100604534149, -0.04494849219918251, 0.008549570105969906, -0.4031069874763489, 0.15899312496185303, -0.9048752784729004, 0.8197560906410217, -0.5077735185623169, -1.5105398893356323, 0.4640621840953827, -1.7604684829711914, 0.29988113045692444, -1.141051173210144, -0.019194576889276505, 0.9295248985290527, 1.0441814661026, -0.7964407205581665, 0.36443284153938293, 0.6745293140411377, -0.20365476608276367, -1.3253731727600098, -1.3864742517471313, 0.3497776985168457, 0.20735985040664673, 0.8721945285797119, -0.47792181372642517, -0.17377561330795288], [0.06034566089510918, -1.3062790632247925, -0.727202296257019, -0.6286082863807678, 0.05900321528315544, -0.10602696985006332, 0.18886932730674744, -1.3105956315994263, -0.03119461052119732, -0.767756462097168, -0.25061604380607605, 0.7888616323471069, 0.11557155847549438, 1.7716084718704224, -0.2174633890390396, -1.3455051183700562, 1.3307406902313232, 0.8276476263999939, 2.0331871509552, 0.950128972530365, 2.4858744144439697, -0.3094899654388428, -0.3069223165512085, -0.7535107135772705, 0.07473520934581757, 0.7467119097709656, 0.7018478512763977, 1.4897983074188232, -0.3342802822589874, 1.5536587238311768, 0.9526747465133667, -0.8368034362792969, -2.2447047233581543, -1.6810916662216187, 0.12129025161266327, -1.2116153240203857, -0.42182475328445435, 0.6750769019126892, -0.5484822988510132, 0.2890946865081787, -0.05480044335126877, 0.5622469782829285, 1.1936235427856445, 0.888476550579071, -0.6345211863517761, -0.9198790192604065, 0.1124320924282074, -1.0812571048736572, -0.1896621733903885, 0.8055906295776367, -0.2270595282316208, -0.8926184773445129, 1.1032204627990723, -1.3108234405517578, -0.15297307074069977, -0.7206860780715942, -0.212494358420372, -0.08742699772119522, 0.3681841790676117, 0.26450595259666443, -0.3151041269302368, -0.2393004149198532, 0.418661504983902, -0.3896137773990631, -1.363318920135498, -0.16608110070228577, -1.2200937271118164, -0.700789749622345, 0.5512447953224182, 1.5332262516021729, 0.9689926505088806, -1.257097601890564, -0.0347101055085659, -0.38060232996940613, -0.9403342008590698, -1.173184871673584, -0.8810267448425293, -0.6822088956832886, 2.7281999588012695, 0.2197076827287674, 1.9446089267730713, -0.8378973603248596, -1.4269729852676392, -0.9784367084503174, -0.026791604235768318, -0.17805713415145874, 1.241563081741333, 1.4043762683868408, -1.2541707754135132, -2.1647846698760986, 0.9364403486251831, 2.0180718898773193, -0.5987328290939331, -2.394218921661377, -0.5041545033454895, -1.268480896949768, 0.4946979582309723, -1.149071455001831, -1.2516005039215088, 0.1681351661682129, -0.4055248200893402, 0.5364227294921875, -1.450777530670166, -2.1959750652313232, 0.24212661385536194, -0.12139751762151718, -1.1529626846313477, 1.0977689027786255, -0.2140946090221405, 0.29864513874053955, 2.1887574195861816, 1.2049672603607178, 0.8377155065536499, -0.015338817611336708, -0.6705856919288635, 0.771377444267273, -1.4752779006958008, -1.285398006439209, -0.21813002228736877, -0.6292340755462646, -1.6297794580459595, 0.5799867510795593, -0.19412511587142944, -0.8105382323265076, 1.4738372564315796, -0.5542325973510742, 0.13571622967720032, -1.3789188861846924], [-1.8909105062484741, 0.7637394666671753, 2.2420547008514404, -0.6251180171966553, 0.6588319540023804, -0.27806299924850464, -0.3021336793899536, 0.2812468409538269, 2.469237804412842, 0.23772719502449036, -0.5511268973350525, 0.0018144453642889857, -0.5578483939170837, 1.090658187866211, -0.671823263168335, -0.5600232481956482, 1.435182809829712, -1.402191162109375, -1.404395580291748, -3.232621431350708, 1.4420312643051147, -0.01926976442337036, -0.7567192912101746, -1.5597885847091675, -1.7521121501922607, -0.5386849045753479, 0.5072098970413208, -1.080540418624878, 0.8285738229751587, 1.634223222732544, 0.8810492157936096, 0.7353432774543762, -0.20962105691432953, -0.5931849479675293, -0.5703744888305664, 0.49629828333854675, 1.4295109510421753, -0.5925074219703674, -0.9006754755973816, -0.31588679552078247, -0.6983896493911743, 0.33156535029411316, -1.0157747268676758, -0.49633321166038513, 1.9490406513214111, -0.4548678994178772, 0.8502278923988342, 0.4844950735569, -1.0681933164596558, 0.23255346715450287, -1.234901785850525, 1.6031984090805054, -0.7550888657569885, 0.04174179583787918, 0.8952399492263794, -1.4088255167007446, 0.9573739767074585, 0.720250129699707, -1.7626984119415283, 0.9284008741378784, -0.8565475344657898, -1.3182406425476074, 1.116366982460022, -0.7419258952140808, 0.43281298875808716, -0.7316697835922241, -1.4774335622787476, -0.011686409823596478, 1.149385929107666, 0.4385707378387451, 0.7742685675621033, 0.5695416331291199, -1.0316671133041382, 0.45252725481987, -0.5448449850082397, 0.9223396182060242, 0.2888718545436859, -0.01990761235356331, -1.5897800922393799, 0.6558716893196106, 0.13984107971191406, 0.3776554763317108, -0.09266462922096252, -1.607864260673523, 1.6700294017791748, 1.3729392290115356, 0.4338246285915375, 0.2978156805038452, -1.4824377298355103, -0.8279744982719421, 1.8178091049194336, -0.6251817941665649, -0.6662032604217529, 1.5281269550323486, 0.6688340306282043, 0.39148151874542236, -0.3747066557407379, -1.657883882522583, -0.7768533229827881, 1.5049216747283936, -1.024675965309143, -1.0706920623779297, 0.03537419065833092, 0.24088633060455322, -1.0406006574630737, 0.6572099924087524, -0.6767301559448242, 1.1097081899642944, -0.3367990255355835, 0.7007246613502502, -0.45173966884613037, 1.1560837030410767, 1.912641167640686, 0.03710845485329628, 0.20489385724067688, -0.19862128794193268, -0.9140001535415649, 0.8785304427146912, -0.6877099871635437, 0.7117132544517517, 0.44381746649742126, 0.6567617654800415, 1.0017188787460327, -1.1566390991210938, -1.6654537916183472, -0.8799634575843811, 0.21948005259037018, -0.5568313002586365], [0.058742955327034, -0.4038262367248535, -1.0667108297348022, -1.4504667520523071, -0.2468983680009842, 1.1879850625991821, 0.6416922211647034, -0.937882125377655, 0.8575531840324402, 0.13458681106567383, 1.3405128717422485, -1.0296694040298462, -0.8325653076171875, 0.15641257166862488, 1.4076998233795166, 0.07911165803670883, -1.6943697929382324, 0.22322140634059906, 0.13599303364753723, 1.2014156579971313, -2.457549810409546, -0.14830993115901947, -0.33023712038993835, -0.6775620579719543, 0.518757164478302, 1.6554285287857056, -0.05624011158943176, -0.19774043560028076, -0.23232963681221008, -1.0538413524627686, -1.3869640827178955, -0.1551792025566101, -0.6623595356941223, -0.03995984420180321, 0.1049443930387497, -2.483935594558716, -0.05690791457891464, 0.2989574372768402, -0.5116718411445618, -0.5901382565498352, -0.7631017565727234, 0.6613089442253113, 1.6612917184829712, 1.1826717853546143, -1.1402047872543335, 1.443457841873169, 0.3651772141456604, 0.8265838623046875, -0.331373393535614, 0.5272417068481445, -0.372480571269989, -0.627297043800354, -1.3676817417144775, -0.28518742322921753, -0.36141547560691833, 2.0146703720092773, -0.32485294342041016, -0.14393861591815948, 0.27033036947250366, -1.552068829536438, 0.5669229626655579, -1.9690349102020264, -0.7021369338035583, -0.6851223111152649, 0.1841958463191986, 0.42077505588531494, 0.8115569353103638, -1.4758665561676025, 1.9659826755523682, -0.8685801029205322, 0.4996902346611023, -0.26704058051109314, 0.46543124318122864, -1.2950305938720703, -0.38661491870880127, 0.7368629574775696, 2.0005836486816406, 0.4770875573158264, 0.838066041469574, -0.5924615263938904, 0.12142474204301834, -1.5816506147384644, -1.4755988121032715, 0.9873659014701843, 0.8642191290855408, -1.093122959136963, -1.3332173824310303, -0.358757883310318, -1.2441977262496948, 1.7846800088882446, -0.40415993332862854, -0.9403076171875, -1.4552606344223022, 0.40946799516677856, 0.33528637886047363, 0.3629525899887085, -0.263871967792511, -0.26769423484802246, -1.3016353845596313, 0.08633892238140106, 0.4925484359264374, 0.059752609580755234, -1.0236014127731323, 0.346367746591568, -0.44399482011795044, 1.7481062412261963, -0.7348645329475403, -0.30702710151672363, -0.2627873718738556, 0.9084323644638062, -0.019290171563625336, -1.253319263458252, 0.06021939963102341, 0.07500359416007996, 0.0049058725126087666, -0.5490778088569641, -1.8673650026321411, -1.7878577709197998, 1.0878393650054932, -0.7766659259796143, -0.24879276752471924, -0.27411726117134094, 0.7989521026611328, -1.7271957397460938, -0.5689224600791931, -0.6331154108047485, -0.33027026057243347, -1.1473232507705688], [0.9277539849281311, 1.3373302221298218, 0.5755983591079712, -0.3847840130329132, -0.2610635757446289, -0.019707513973116875, -0.4253825545310974, -0.3127245306968689, 0.01911928690969944, 0.5564404726028442, 2.1705007553100586, -0.01627831533551216, -0.7415497899055481, -1.1711183786392212, 0.7508295774459839, 1.521704912185669, 1.358393907546997, -1.623783826828003, -0.5902991890907288, 0.22588300704956055, 1.157780408859253, -0.4543797969818115, -1.11432945728302, -0.6341018080711365, -0.11813497543334961, 0.7861631512641907, 0.7934750318527222, -0.36910560727119446, -0.4332933723926544, 0.5493459701538086, -0.18061333894729614, 0.897750973701477, -1.8356328010559082, 0.052434712648391724, 0.01880825310945511, 0.23549266159534454, 0.43505436182022095, 0.4108520448207855, -0.62404465675354, 1.281685471534729, 0.13348713517189026, -0.17327065765857697, -1.2616938352584839, 0.45343080163002014, -0.91579669713974, 0.4698035418987274, 1.227349877357483, -0.16929754614830017, 2.6947991847991943, -0.5377587080001831, 1.1181813478469849, -0.9940308928489685, -1.102534532546997, 0.7961552143096924, 0.9934210181236267, 0.5571804642677307, -0.7072086334228516, -0.9377694129943848, -0.967110812664032, -0.13574156165122986, 0.09134559333324432, -0.4833854138851166, -0.5909325480461121, 1.1685901880264282, -0.5711098909378052, -1.6061605215072632, -0.46634653210639954, -0.9391673803329468, 1.1744544506072998, 2.8073065280914307, -1.3771523237228394, -0.12557263672351837, -0.03167955204844475, -0.8784729242324829, -0.02359846793115139, 0.6769130229949951, -0.017462648451328278, -0.33907392621040344, 0.09446488320827484, 0.6554194092750549, -0.5303104519844055, -0.4453590214252472, 0.5869150757789612, -0.2231680452823639, -0.48398861289024353, -0.7217719554901123, 0.13365308940410614, 0.5335723757743835, 1.671256422996521, 0.09163729101419449, -0.5011847019195557, 0.17564274370670319, 1.1066409349441528, -0.7137181758880615, 0.21125657856464386, -0.8968459367752075, -0.1372002810239792, -1.0089937448501587, 0.8126571774482727, -0.154526486992836, -0.9171627759933472, -0.826965868473053, 0.3534483313560486, 1.374806523323059, 0.2512015998363495, -1.431106686592102, 0.8771611452102661, 0.538564920425415, -0.24244628846645355, -0.9265092611312866, -0.9779388308525085, -0.11994343996047974, 0.5479493141174316, 0.014184889383614063, 0.43360474705696106, 0.311468243598938, 2.126701831817627, 0.8688718676567078, 1.3187364339828491, 0.5039469599723816, -0.22488929331302643, -0.366731733083725, 0.15890009701251984, 0.12520650029182434, -0.3342887759208679, -2.0908076763153076, -0.44595035910606384, -0.01925724931061268], [-1.0499770641326904, 1.6963063478469849, 0.24715310335159302, -0.9224881529808044, 1.6368169784545898, 0.4176628887653351, 0.1417781114578247, 0.5614618062973022, 0.3986387848854065, 0.3924444615840912, 1.1798170804977417, -0.0609096921980381, 1.2814337015151978, 1.1367355585098267, -1.211159110069275, -1.4937667846679688, -0.43998992443084717, 2.1880900859832764, 1.0143015384674072, -0.711539089679718, 0.33332470059394836, 0.28677675127983093, 0.583763837814331, -0.22375917434692383, 0.8214386105537415, 1.248734474182129, 1.4887808561325073, 0.30074289441108704, 0.11891590058803558, 2.4067091941833496, 1.0541470050811768, 0.21252359449863434, 0.5155863761901855, 0.47661933302879333, -1.4936254024505615, 2.1472716331481934, 0.4291832745075226, 0.17026826739311218, 0.7184536457061768, -1.750259280204773, 0.32671788334846497, 0.058488063514232635, -0.32937905192375183, -0.9369823932647705, -2.4020633697509766, -0.3962878882884979, -1.0757856369018555, -0.38160043954849243, -0.7921841740608215, -0.21691547334194183, -0.6216343641281128, 0.2593580484390259, -0.03979204595088959, 0.6147798299789429, 0.4119711220264435, 1.566434383392334, -0.3327874541282654, -0.6044301986694336, 0.6612246632575989, -0.7761094570159912, 1.6015750169754028, -1.9979803562164307, 0.2166014462709427, -2.147840738296509, -0.28469547629356384, 0.9306268095970154, 0.6950960159301758, 0.8090885877609253, -0.4224577844142914, -0.5226364731788635, 1.6306005716323853, 0.1486804187297821, -0.006226575002074242, 0.7010831832885742, 0.008939986117184162, 1.0418415069580078, -0.11728273332118988, 0.698196530342102, -0.2600349485874176, -0.9658511281013489, 0.7400807738304138, 0.3059183657169342, 0.8573837876319885, -0.48589545488357544, -0.5849697589874268, 0.25877097249031067, -0.7306346297264099, -0.05925925821065903, 0.16116952896118164, -0.6337944269180298, 0.6892691850662231, -1.1009083986282349, -1.6133676767349243, -0.17872078716754913, 0.8606353998184204, 1.1641590595245361, 0.01852906309068203, 1.0062305927276611, 0.6344863772392273, -1.76161789894104, -0.602837085723877, -0.21655593812465668, 0.584708034992218, -0.01880417764186859, -0.6099315285682678, -0.05320535600185394, -2.3045153617858887, 0.12215150147676468, 1.386725664138794, 0.7080673575401306, 1.5279462337493896, -0.6659454703330994, -0.1862097829580307, 1.536380410194397, -0.5438223481178284, 0.7502274513244629, -0.2998594045639038, 0.04225912690162659, 0.2847990393638611, 1.4229798316955566, -0.4294343590736389, -0.7995113134384155, 0.039898671209812164, -0.03388811647891998, -0.5893085598945618, 0.7990780472755432, -0.2761279046535492, -1.166264295578003], [-0.3806115984916687, -0.2580738067626953, 0.7280280590057373, -1.0822527408599854, -1.2368414402008057, 2.17443585395813, -0.21548451483249664, -0.13315311074256897, 0.6780188679695129, 1.4004579782485962, 0.6110265851020813, -0.42755040526390076, -0.5014044046401978, -1.154394507408142, -0.6847654581069946, -0.9981098771095276, -0.5164142847061157, -0.4073157012462616, -0.047148074954748154, -0.36684080958366394, 1.3922532796859741, -0.5627878308296204, 0.4331890940666199, 0.4794539213180542, 0.2305862307548523, 0.7924861311912537, -0.3393392860889435, -0.8371546864509583, 0.23895764350891113, 0.3554460406303406, 1.005152940750122, 1.3193105459213257, 0.2858189046382904, 1.9778258800506592, 0.9762898683547974, -1.4760751724243164, -0.06804508715867996, -0.5149965286254883, -0.006155199371278286, -0.5272099375724792, -0.6238279342651367, 0.22612734138965607, 1.1099820137023926, 0.38383620977401733, 0.5005825161933899, -1.083385944366455, -1.005989909172058, -0.24207767844200134, -0.015167768113315105, -0.38818100094795227, -0.992631733417511, -1.938366174697876, -0.3869195580482483, -0.6126707196235657, -0.6698983311653137, -0.829291045665741, 0.5820607542991638, 0.37901875376701355, 0.4799312651157379, -1.1123526096343994, 0.3955884873867035, -0.8853632807731628, 0.5631686449050903, -0.7041202783584595, 0.16591084003448486, -0.5790864825248718, 0.164534330368042, -2.3276114463806152, 0.21267621219158173, 0.17056460678577423, 0.26230132579803467, 0.016691109165549278, -1.2830694913864136, 0.5107619166374207, 0.7500607967376709, 0.8150858879089355, 2.0399327278137207, -0.2105753868818283, 0.015922153368592262, 1.3369884490966797, -1.2537158727645874, -0.3518948256969452, -1.189415454864502, 0.14891178905963898, -0.47345152497291565, 0.6333870887756348, -1.1831321716308594, -1.4168541431427002, -0.11777616292238235, -0.7440864443778992, -1.2632575035095215, 0.7722081542015076, -0.6330445408821106, 0.7148763537406921, -1.9543002843856812, -2.142690896987915, 0.6459258794784546, 1.3508999347686768, 1.477076530456543, -0.3516960144042969, 1.2444190979003906, 1.6866803169250488, 0.14506842195987701, 0.8492352366447449, -0.5045763850212097, -0.9501594305038452, 0.09507464617490768, 0.773736834526062, 0.5999709963798523, 0.6945662498474121, -0.6272123456001282, 0.4563405513763428, -0.5663946866989136, 0.09548069536685944, 0.16228361427783966, -0.7368833422660828, 1.030777096748352, -0.18743561208248138, 0.7371134757995605, 0.04637886583805084, 1.0207302570343018, -1.064548373222351, -0.883299708366394, -0.7848289608955383, 0.2137833684682846, 0.40963372588157654, 0.1933724284172058, -0.5510937571525574], [0.4793083667755127, 0.1813347488641739, -1.0254274606704712, 0.9788328409194946, -0.15472015738487244, 0.39959225058555603, -0.17551536858081818, -0.07024946063756943, 0.5026997923851013, 0.09866338223218918, -1.5425841808319092, 0.33607593178749084, 0.6546251177787781, 0.8743084669113159, -1.063414216041565, -0.2330031394958496, -1.3618043661117554, 0.8354871273040771, -0.10221260786056519, 0.22990216314792633, 0.13335967063903809, -0.6751506924629211, 0.5038847923278809, -1.6720197200775146, 1.026198148727417, 0.07792795449495316, -0.9016307592391968, 0.3419819474220276, -0.8093050718307495, 1.0881803035736084, -0.6666717529296875, -1.058628797531128, 0.9368295073509216, -0.7447307109832764, -0.4739742875099182, -0.7999778389930725, -0.40151435136795044, -1.488214135169983, -0.5309045314788818, 1.7374231815338135, -0.5808452367782593, 2.190237045288086, 1.5230330228805542, 1.6293249130249023, 1.236429214477539, 1.9989864826202393, -0.2501783072948456, -1.3347963094711304, 1.2909815311431885, -1.0727828741073608, -1.2736506462097168, 1.6076338291168213, -0.04180927202105522, -0.10131599009037018, 0.777655303478241, 0.7036446928977966, 0.4382518231868744, 0.9317468404769897, -0.03730305656790733, 0.7717049717903137, -0.14952366054058075, -0.45208147168159485, -0.8616507053375244, -0.3846520483493805, 0.7484858632087708, 0.9603344798088074, -1.1362909078598022, -0.8865826725959778, 2.0460124015808105, -1.38486647605896, -0.03759128972887993, -0.6813288927078247, 0.21676243841648102, 0.06461700052022934, -2.5902600288391113, 0.36098170280456543, -3.2438533306121826, -0.0330827496945858, 0.26431500911712646, 0.24914802610874176, -0.28352031111717224, 0.5883920192718506, -1.4311201572418213, -1.5162791013717651, 0.4601078927516937, -0.38496458530426025, -0.7239009737968445, 1.372241735458374, 0.4380878508090973, -0.6789109110832214, 0.4855101406574249, 0.5060569047927856, -0.09801024198532104, 1.2151012420654297, -0.28658002614974976, -1.0619922876358032, 0.5191508531570435, 0.8678157925605774, -0.8607243895530701, 0.7606489658355713, 0.9516905546188354, -1.040278673171997, -0.09834836423397064, -0.06700088828802109, 1.4390952587127686, -0.5981593132019043, 0.32496413588523865, 0.11865347623825073, -0.7594075202941895, -0.3180473744869232, -0.36862727999687195, 0.7099533677101135, -0.6558037996292114, -0.464385986328125, -1.1291396617889404, -0.5587353706359863, -0.5972458124160767, 0.10346557199954987, 0.724654495716095, 1.7280066013336182, 0.126506045460701, 1.0644469261169434, 1.0648057460784912, 0.10344705730676651, -0.6030572056770325, 1.6517436504364014, 1.5994176864624023, -0.019626393914222717], [-1.284563422203064, -0.3433539867401123, -0.9006718397140503, 0.2334093302488327, -1.1540923118591309, -1.9253367185592651, -1.33309805393219, 0.7741640210151672, 0.2685191035270691, -2.4715144634246826, 0.7123100757598877, -0.024586278945207596, 0.04104315862059593, 0.4485657215118408, -0.1410658061504364, -2.7037765979766846, -0.22415074706077576, 0.11322830617427826, 0.2699629068374634, -0.140615776181221, 1.337026596069336, 0.20820185542106628, -0.9405038356781006, 0.7217261791229248, -2.0937700271606445, 1.030724287033081, -1.3019347190856934, -0.8979656100273132, 0.2994645833969116, 1.1338621377944946, -0.6136900186538696, -1.668349027633667, 0.7653283476829529, 1.6121420860290527, -1.2596644163131714, -0.23200726509094238, 0.3291756212711334, 0.20218956470489502, -0.7553549408912659, 0.03761417046189308, -0.49358728528022766, -1.8544132709503174, 1.0467621088027954, -0.6144741773605347, -1.336863398551941, -0.13268832862377167, -0.2533799409866333, 0.6646499037742615, -0.39983275532722473, 0.09960505366325378, -0.24651625752449036, -1.2496507167816162, 0.4563809633255005, -2.476741313934326, 0.49007460474967957, 0.2839103043079376, 2.7656280994415283, -0.416856974363327, -1.4452025890350342, -0.05695941671729088, -0.33483049273490906, -0.8006356358528137, 0.6317355036735535, 0.5605005025863647, 0.14918695390224457, -1.5303171873092651, 1.6265513896942139, -1.9220058917999268, 1.7662968635559082, -1.5301867723464966, 1.76669442653656, -1.1058154106140137, 0.11412874609231949, -1.487765908241272, -0.8135301470756531, -0.8734153509140015, -1.3598415851593018, 0.13602620363235474, -0.9929325580596924, 0.0502297542989254, -0.3292441666126251, 0.35870426893234253, 2.0220553874969482, 0.3340425491333008, 0.5265278220176697, -1.4532254934310913, -0.749626100063324, 0.779941976070404, 0.15392017364501953, -0.6010417342185974, -0.17732885479927063, -0.339860737323761, 0.10595113784074783, -0.45342934131622314, 0.8654180765151978, -0.8994373083114624, -1.6407959461212158, -0.6348158717155457, 0.2008083462715149, 1.5489552021026611, -0.49181050062179565, 0.9037268161773682, -0.0777563750743866, -0.8173051476478577, 0.011837909929454327, 0.6910481452941895, 1.3194208145141602, 0.6106901168823242, 1.3707597255706787, -0.20024409890174866, -0.3889226019382477, -1.5041722059249878, 0.7041988968849182, 0.8445293307304382, 0.9288464188575745, -1.6451274156570435, 0.45093923807144165, -0.8462088704109192, 1.4634579420089722, -0.12596242129802704, 1.3833168745040894, -0.0287947878241539, 0.12843681871891022, 1.5854860544204712, -0.5844711661338806, -1.4377917051315308, -0.43659210205078125, -0.20028497278690338], [-2.2717268466949463, -0.4343350827693939, -0.2261786311864853, 1.290238857269287, 1.6794651746749878, 0.6603900790214539, -1.0141332149505615, 0.6193228960037231, -0.7400439381599426, -0.33201584219932556, -0.8464553356170654, -0.9220383763313293, -0.7266064882278442, -0.9794921875, 0.03141719847917557, 0.7162400484085083, -0.23886531591415405, -0.41718757152557373, -1.4348516464233398, -0.2090531587600708, -0.1419266313314438, -2.3409905433654785, 0.6885899901390076, -0.486263245344162, 0.24745675921440125, 0.0773266926407814, -1.4576120376586914, -0.3982316851615906, 0.5072929263114929, 1.5461241006851196, -0.14248280227184296, -0.4542100429534912, -0.7278274297714233, 1.4064719676971436, 0.5960618257522583, -0.18601062893867493, 0.7451125979423523, -0.08698935806751251, 0.9423922896385193, -0.12461002916097641, 0.49670878052711487, -2.7953548431396484, 0.2197953462600708, 1.3046268224716187, 0.40128934383392334, -0.17589667439460754, 0.08781186491250992, -0.3795895278453827, 0.3895346224308014, -0.32309195399284363, -0.6962809562683105, 0.5360589027404785, -1.2265676259994507, -0.04182356595993042, -0.08442018181085587, -0.10123027116060257, 0.253518283367157, -1.0356439352035522, 1.120615839958191, -0.4661605954170227, 0.909939169883728, -0.6232713460922241, 0.26377612352371216, 0.534061074256897, 1.204946756362915, 0.8895784616470337, -0.41770267486572266, -0.33395957946777344, 0.7426832318305969, -0.4146062731742859, 0.7682214379310608, 0.807111918926239, 0.6041626930236816, -0.6690198183059692, -0.25842946767807007, -1.100680947303772, 1.9586819410324097, -0.004758263472467661, -0.1764964759349823, -0.05400367081165314, 0.5927048325538635, -1.27874755859375, -1.177893877029419, 1.1775825023651123, -0.17295967042446136, 0.9154515862464905, -0.3273262083530426, -1.8775712251663208, -0.12321257591247559, -1.6179012060165405, -0.5270315408706665, -0.8541695475578308, -0.5251079797744751, 0.4607515037059784, -1.1275720596313477, 1.2121087312698364, 0.4176100492477417, -2.068014144897461, -0.24706558883190155, -1.586646318435669, 0.7183360457420349, 0.2989931106567383, 0.1659669429063797, -0.7881678938865662, -0.35111790895462036, 1.3295577764511108, -1.2915732860565186, -0.19194243848323822, -1.1369808912277222, -0.3281199336051941, 0.5850030183792114, -0.844680905342102, -0.5581789016723633, -0.2731876075267792, -0.7838815450668335, -1.6797317266464233, -2.0109596252441406, 0.658058762550354, -1.1706775426864624, -0.2991112172603607, 0.1803206503391266, 1.8252850770950317, -0.3615230321884155, 1.2420331239700317, 0.36209937930107117, -1.6360009908676147, 0.09723634272813797, -0.7740774750709534], [-0.3741815984249115, -0.5679690837860107, -1.9060314893722534, -0.15917660295963287, -0.32239922881126404, 0.5265399217605591, -0.5610138177871704, 1.1094056367874146, -0.8470280170440674, 0.41369980573654175, 0.6593052744865417, 0.047689229249954224, 0.5215341448783875, -1.0103703737258911, -0.9887190461158752, 0.8050258159637451, -1.9584908485412598, -0.33900195360183716, -0.49996262788772583, -0.43114200234413147, -0.23585909605026245, -0.11306007951498032, -0.26038461923599243, 1.3310242891311646, 0.20439589023590088, 0.955818235874176, -0.5896473526954651, 0.843145489692688, -0.6169503331184387, -1.1615586280822754, -0.2706218957901001, 1.1971179246902466, 1.3648179769515991, -1.223738431930542, -1.1516131162643433, 1.01773202419281, -1.1323527097702026, -1.7116339206695557, 1.4569673538208008, 0.5053097009658813, 0.5334877967834473, 0.5425856113433838, 1.6424986124038696, -2.7726504802703857, 1.219046950340271, 1.1537232398986816, -0.31108078360557556, -0.20710204541683197, -1.1556892395019531, -2.7770512104034424, 0.6501198410987854, 0.53825443983078, -0.27705731987953186, 0.18855416774749756, 1.7502950429916382, 0.9271353483200073, 1.5387041568756104, -0.04538014531135559, -0.8966932892799377, 0.5195623636245728, 0.26145440340042114, -1.1131088733673096, 2.0641918182373047, 0.6156457662582397, -1.022802472114563, 0.09259095042943954, 1.7365837097167969, -0.9799520373344421, -1.5234167575836182, -2.1805801391601562, 1.2457057237625122, 0.14201894402503967, 0.03538624942302704, -1.0045361518859863, 0.8034607768058777, -0.47127974033355713, -0.6121159195899963, 0.40018993616104126, 0.516309380531311, 0.12084908038377762, 0.3778367042541504, -0.34866610169410706, -0.10367278009653091, 0.3575800657272339, -0.07532544434070587, -0.44401705265045166, -0.138475701212883, -1.4383295774459839, -0.3358980417251587, 1.5064905881881714, 0.36969009041786194, 1.1120413541793823, 0.3638700842857361, -0.9415445923805237, -0.7737846970558167, -0.8311311602592468, -1.1684263944625854, 0.533306896686554, 1.136884331703186, -1.6886407136917114, -0.8425474166870117, 0.7085726857185364, -0.5120551586151123, 0.41842466592788696, 0.10247939825057983, 1.0874695777893066, -0.00880442839115858, -0.45125699043273926, 1.230630874633789, -0.06896205991506577, 0.9533438086509705, 1.042893648147583, -1.351981282234192, -0.05737641453742981, 0.03892437368631363, -0.23152391612529755, -0.4546832740306854, -1.551170825958252, -0.8137653470039368, 1.06755793094635, 1.325490117073059, -1.1467911005020142, -0.08346344530582428, 0.7003480195999146, 2.143639087677002, -0.6034548282623291, -1.1870321035385132, -0.2321334332227707]], [[-0.5725002884864807, 0.6395916938781738, 0.15249383449554443, -0.2652405798435211, 0.5688409209251404, -0.5046730637550354, 1.5183918476104736, -0.15063300728797913, 0.5235198736190796, -1.0985617637634277, -0.6840851902961731, -0.19839751720428467, 0.020620597526431084, -0.9430146813392639, 0.6383148431777954, 0.2650532126426697, 1.5296709537506104, 0.1435292661190033, 1.1976981163024902, -1.622244954109192, -0.2018374651670456, -1.7942590713500977, 0.15418177843093872, 0.3158662021160126, -1.9610038995742798, 0.5733601450920105, -0.8911686539649963, 0.2731156051158905, -0.10539348423480988, 0.5534077286720276, -0.3256268799304962, 0.3604715168476105, 1.5341215133666992, 0.08529774099588394, 0.04456669092178345, 1.4496219158172607, 1.5537924766540527, 0.6789305806159973, 0.3509331941604614, 0.021555280312895775, -0.5258382558822632, -0.1718006283044815, -0.5207865238189697, 0.2662879526615143, 2.0553812980651855, 1.3361196517944336, 1.0930200815200806, -0.18911445140838623, -0.9257494211196899, -1.6579930782318115, 0.7885884046554565, 0.17095668613910675, 1.4963529109954834, -0.017588350921869278, -0.42764005064964294, -0.2020416557788849, -0.7439334392547607, -0.46844175457954407, -0.34892308712005615, 0.1420673131942749, 0.9648298621177673, 1.1185334920883179, 1.7594377994537354, -1.3533363342285156, 0.5217671990394592, -1.4386365413665771, -0.7225812077522278, -0.44797831773757935, -1.8408247232437134, -0.10301349312067032, 2.224648952484131, -1.945236086845398, -0.7608911991119385, 1.0962374210357666, -0.017515525221824646, -0.42652449011802673, -1.5620092153549194, 0.1670488715171814, 1.1556687355041504, 1.1995594501495361, 0.40997931361198425, 0.1096080020070076, 0.02481924556195736, -0.23215018212795258, 0.017426177859306335, -0.7871604561805725, -0.6921052932739258, -1.2155364751815796, -0.02292201854288578, -0.1793200969696045, -0.4541158378124237, 1.8522968292236328, -0.45183065533638, -0.5290345549583435, 0.9884675741195679, 0.40675443410873413, 0.5142796039581299, 0.5905113220214844, 0.43933263421058655, -1.8784137964248657, 0.17790858447551727, 0.8469825983047485, -1.0026190280914307, 1.7375472784042358, -0.16251946985721588, -0.6198983788490295, -0.6669979691505432, -1.1246713399887085, -0.31128108501434326, 0.05827844515442848, 0.2611726224422455, -0.5125175714492798, -0.14873464405536652, 1.4195307493209839, -1.938218116760254, 2.092294931411743, -2.040945529937744, 1.7400729656219482, -1.037152886390686, 0.14813832938671112, 0.6402760148048401, 0.06326203048229218, -0.016748854890465736, 0.2552996873855591, -0.9483239054679871, -1.131272792816162, -1.268911361694336, 0.4154469072818756], [0.5989309549331665, -1.3577195405960083, 1.8037089109420776, -1.8154939413070679, -0.5654248595237732, -0.8218743205070496, -0.5792039632797241, -0.9664804339408875, -1.608961582183838, -0.3450433313846588, -0.4417625367641449, 1.5557447671890259, 0.647538423538208, -0.8657087683677673, 0.7043038606643677, 0.9842328429222107, 1.173046350479126, 0.32677435874938965, 1.139050006866455, 1.2549176216125488, 0.1687604784965515, -1.1029809713363647, -0.9734249711036682, -0.41223910450935364, 1.1328940391540527, -1.2773005962371826, -1.233566403388977, -1.9253394603729248, -0.18566375970840454, -1.0292493104934692, -1.1900185346603394, 2.0413429737091064, 0.39476823806762695, -0.22881364822387695, -0.3987491726875305, -0.22706468403339386, 0.9133926033973694, 1.3552132844924927, -1.0289359092712402, 0.9521804451942444, -0.5953438878059387, -0.7294743657112122, -0.41448095440864563, 1.1503617763519287, -0.14707733690738678, 1.0795536041259766, 0.6698659658432007, 0.9020075798034668, 0.7552838325500488, -1.5707924365997314, -0.4151710569858551, -0.4087394177913666, -1.3533207178115845, -0.4162290692329407, 1.1335686445236206, -0.013859988190233707, -1.7692346572875977, -0.7822070121765137, 0.7143632769584656, -1.096058964729309, -0.49491435289382935, 0.10041302442550659, -1.4172247648239136, 1.8269141912460327, -2.487338066101074, -0.2882460355758667, -0.0616760216653347, 0.9137367010116577, 0.8072184920310974, -2.0488948822021484, -0.3824201226234436, 0.8617417812347412, 0.4777831435203552, -0.9404864311218262, 0.5212022662162781, -1.4098007678985596, 0.2977103888988495, 0.05323812738060951, -0.17830753326416016, -1.188788652420044, -0.12000267207622528, -0.7117195725440979, -1.0410884618759155, -0.010477602481842041, 1.0891227722167969, -0.3465997874736786, 0.29176196455955505, -2.1926727294921875, 0.14800795912742615, 0.1712264120578766, 2.2608346939086914, -0.8364176154136658, -0.30967432260513306, -0.13957448303699493, 0.22881609201431274, -1.763411045074463, -1.1690993309020996, -0.7633727788925171, -1.0196771621704102, 0.1950637698173523, 0.2848065495491028, -2.1681008338928223, -1.328937292098999, 0.2175351083278656, -0.4090108871459961, -2.2137725353240967, 1.3087399005889893, 0.7493249773979187, -0.11088462173938751, -1.3200002908706665, 0.048159677535295486, 1.2671297788619995, 0.33623766899108887, 0.5133148431777954, 0.47000062465667725, -0.6166223287582397, 0.34347304701805115, 0.325053334236145, -1.6384514570236206, -0.31315070390701294, -0.13687697052955627, 1.1315621137619019, 1.1801259517669678, 0.755716860294342, -1.1660704612731934, 0.47198617458343506, -0.5446887016296387, 1.0368598699569702], [-0.23770339787006378, -0.643231213092804, -1.4298386573791504, 1.6053677797317505, 0.6570138335227966, -0.2081068754196167, -1.2248281240463257, -0.5361890196800232, -0.7097491025924683, 1.385135293006897, 1.542972207069397, 1.1995905637741089, -0.9319091439247131, 0.07638399302959442, 0.40299659967422485, 1.2625786066055298, 0.22552300989627838, 1.3079867362976074, 0.5710467100143433, -1.1771965026855469, -0.8054138422012329, -0.1909012645483017, 0.25783923268318176, -0.9101435542106628, -0.3782863914966583, -0.26183804869651794, -1.2425787448883057, -0.9438900351524353, -0.07931728661060333, -1.0004605054855347, -0.009510545991361141, -0.10423479229211807, 0.7215794920921326, -0.6794062256813049, 1.9392024278640747, 0.5795093774795532, 0.550150990486145, -0.011649495922029018, -0.03777800127863884, -1.2396408319473267, 0.4917617738246918, -1.6350935697555542, -0.8164727687835693, 0.6072922945022583, 0.046599380671978, -0.9616785645484924, -0.7688961029052734, -0.5921632647514343, -0.27128544449806213, 0.223343163728714, -2.3142130374908447, 0.17473134398460388, 0.5268197655677795, 0.6315581798553467, -0.50040602684021, -0.2959236204624176, -0.38235071301460266, -0.2429480403661728, 0.3520110249519348, -0.9914562702178955, 1.6964970827102661, 0.2454051822423935, -0.6673400402069092, 0.6397612690925598, -1.3822702169418335, -0.9493578672409058, -0.945156455039978, 1.5671875476837158, 0.7230704426765442, 0.04990440979599953, 0.05228878930211067, 1.2463841438293457, -0.8383467197418213, 0.7788567543029785, 0.4870634078979492, -0.7779896855354309, 0.5698527693748474, 1.920518398284912, 0.25940924882888794, 0.5963723659515381, -0.45295560359954834, 0.6071323156356812, -0.4942294955253601, 0.6711921691894531, 0.5953647494316101, 0.10648172348737717, -1.592737078666687, -1.9441347122192383, -0.4840203523635864, -0.46989625692367554, -1.4412450790405273, -1.6082775592803955, -0.2422305941581726, -0.04315375164151192, -1.3310470581054688, 0.6464231610298157, 0.8053038120269775, 1.125725269317627, -2.3031928539276123, -0.04797890782356262, -0.1994129717350006, -0.2715624272823334, -0.0541488379240036, -0.13354803621768951, -0.49518316984176636, 0.6765474081039429, -0.8417556285858154, -1.4168225526809692, 0.7801311612129211, 0.1731923520565033, 1.3174277544021606, -0.29351601004600525, -2.270324945449829, -0.7526471614837646, -3.170257806777954, 0.7945679426193237, -0.5590518116950989, 1.5158859491348267, 0.24141700565814972, -0.27387112379074097, 0.43080365657806396, -1.7186216115951538, 2.577038526535034, 0.7375999689102173, 1.2273677587509155, -1.5048394203186035, 0.16992467641830444, -0.7552562355995178], [-1.7833483219146729, 0.588412344455719, -0.805276095867157, -1.6360825300216675, 0.6748558282852173, -0.7577725648880005, -0.049952346831560135, 0.2254381626844406, -1.5612118244171143, 0.4091368615627289, 0.9405937790870667, 0.6080530285835266, 2.160717010498047, 1.2995851039886475, -1.1407668590545654, 0.5503497123718262, 0.6603304147720337, 0.2026432901620865, -0.19325189292430878, 0.3291890621185303, 1.340923547744751, -0.5616082549095154, -0.9199522733688354, 1.5866233110427856, -0.9270941019058228, -0.04558328539133072, 0.35765689611434937, -0.6324208974838257, 0.5452196598052979, -0.277210533618927, 0.2920595109462738, -0.5656442046165466, 0.27297812700271606, -0.48771795630455017, -0.4287415146827698, -0.48511025309562683, -0.5593127012252808, 0.7278178930282593, 1.0973867177963257, -1.4440770149230957, 0.03771572560071945, -0.11633948981761932, -0.14733107388019562, 0.09456289559602737, 0.561762809753418, 0.2891281843185425, -0.4678719937801361, -0.011555829085409641, -0.3895258903503418, -0.6288477182388306, 0.28262534737586975, 0.6508409380912781, -0.08316915482282639, 0.41968950629234314, -0.627357542514801, 0.8793299198150635, 0.21092283725738525, 0.684941291809082, 1.0673109292984009, 0.463379442691803, -0.12757675349712372, -2.2391791343688965, -1.0677355527877808, 0.34809306263923645, -1.0806456804275513, 1.1464166641235352, 0.8031231164932251, -0.26108962297439575, -0.5187751054763794, -0.3822818696498871, 0.29342004656791687, -1.533624529838562, 0.3343389630317688, 1.5275582075119019, 0.9952265024185181, 1.4568129777908325, 0.01568557694554329, -1.8661425113677979, 0.6553802490234375, 0.1137896403670311, -0.6478800177574158, -0.4586535096168518, -0.8793109655380249, 0.2949727475643158, 0.13758252561092377, 1.2876564264297485, -0.3012067973613739, -1.252175211906433, 0.6182869076728821, -1.2711917161941528, 0.4647214114665985, 0.8558141589164734, -1.6634314060211182, -1.1312503814697266, -0.7758410573005676, 2.197399377822876, 1.083418846130371, 2.0300159454345703, 0.5102919936180115, -0.42684367299079895, 0.5056633949279785, -1.1273581981658936, 1.8864696025848389, -0.27851155400276184, -0.08054209500551224, -1.238196849822998, 1.2193180322647095, -0.40863171219825745, 0.2717653512954712, 0.6074869632720947, 0.753424346446991, -0.309255987405777, 0.08022075891494751, -0.4280596077442169, -0.8552585244178772, 1.075244426727295, -1.8532278537750244, -0.31702035665512085, 0.5728091597557068, 1.0039490461349487, -0.5542454123497009, -1.141619324684143, -1.6486854553222656, 1.6470164060592651, 1.1003168821334839, 0.38644716143608093, 0.6387050747871399, 2.3714070320129395], [0.3997126817703247, 1.0216898918151855, -0.3496631681919098, 0.03092336468398571, 0.6202355623245239, -0.41671597957611084, -0.5178791880607605, -0.3398224413394928, 1.5092397928237915, -0.9397544860839844, 0.5513192415237427, -0.008606499992311, 0.6873704195022583, -0.6541728973388672, 0.2884947955608368, 1.176797866821289, -0.5903239250183105, 1.3094627857208252, 1.1978932619094849, -1.8891633749008179, 0.5218955874443054, -0.033771026879549026, -1.3965113162994385, 1.0391547679901123, -0.985081136226654, 1.070554256439209, 0.5856937766075134, 0.8853113055229187, -0.6753894686698914, 0.36774542927742004, 0.030164405703544617, -0.7275821566581726, 0.01722363568842411, 1.3328830003738403, 0.242047980427742, -1.603463053703308, -1.1192400455474854, -0.569337785243988, -0.8316349387168884, 0.8485398292541504, -1.4407724142074585, 0.235457643866539, 1.4067795276641846, -0.891208291053772, 0.2996666431427002, -1.5836604833602905, -0.3367127776145935, 0.291687548160553, 1.8368782997131348, 1.4948012828826904, 0.6690189838409424, -0.506860613822937, -1.0890510082244873, 0.5826745629310608, -0.46905970573425293, -0.3278440237045288, -0.8612530827522278, -3.195528984069824, -1.6744948625564575, -3.2806131839752197, 1.6199408769607544, -0.5542622804641724, -1.5416547060012817, 1.6238678693771362, -0.8825649619102478, -0.08092091232538223, 0.9982504844665527, -0.16708897054195404, 1.7278430461883545, -0.13571009039878845, -2.1437644958496094, -0.3101878762245178, -0.34616950154304504, 1.6849263906478882, -0.2706463038921356, 1.235730767250061, -0.24861806631088257, -0.37410590052604675, 0.060395728796720505, 0.41343098878860474, 0.5549463033676147, 0.8895481824874878, -0.5122288465499878, 0.08172985911369324, -0.003079708432778716, -2.971346139907837, 1.2676324844360352, -1.518789291381836, -1.1351921558380127, -0.620041012763977, 0.5506871342658997, -0.8953604102134705, -0.20741967856884003, 2.6894376277923584, -0.05842306464910507, -0.035526663064956665, 0.44499921798706055, 1.4906154870986938, 1.5652142763137817, 0.5879684090614319, 0.24016381800174713, -1.0620083808898926, -0.13721877336502075, 0.43684011697769165, 0.20749512314796448, -0.4035069942474365, 0.9642515778541565, 0.5949954390525818, -0.7356957197189331, -1.1108434200286865, 0.47147509455680847, -0.5132167935371399, -0.3590150773525238, 0.669925332069397, 0.5342057347297668, 0.9123018383979797, 0.997934877872467, 0.7177188992500305, 0.052537936717271805, -0.22711633145809174, 2.8739688396453857, 2.2651708126068115, 0.449700266122818, 0.7112902402877808, -0.3230159282684326, -0.019224083051085472, -0.7076782584190369, -0.5788192749023438], [0.6851774454116821, 1.1331902742385864, 2.299671173095703, -1.050024151802063, 0.8028264045715332, -0.9199905395507812, -0.22096772491931915, 0.9934349656105042, 0.993185818195343, 0.9160696864128113, -0.5051433444023132, -0.966256856918335, -1.0549606084823608, 0.022296594455838203, -1.0376487970352173, 0.7361504435539246, 1.3205958604812622, 0.6358805298805237, -1.1687325239181519, -1.150564432144165, -0.27895453572273254, 0.7768778800964355, 0.19975417852401733, 0.5436021089553833, -1.3935801982879639, -1.176105260848999, 0.47428154945373535, -0.5084579586982727, -1.080068588256836, -2.07918381690979, 1.0836336612701416, 0.06681285798549652, -1.0917388200759888, -0.5032937526702881, -0.2852800190448761, -0.19998309016227722, 1.655198097229004, 1.343253254890442, -0.07799271494150162, -1.3510288000106812, -0.49376824498176575, -0.5155573487281799, -0.7590166330337524, -0.8353922963142395, -1.7753831148147583, 0.04828878119587898, -0.4401291310787201, -0.29940614104270935, -0.22299788892269135, 0.04918954148888588, 0.9213495254516602, 1.1078836917877197, 0.8213827013969421, -1.5614573955535889, 0.394873708486557, 0.16145117580890656, 0.16818013787269592, -1.122623085975647, -0.4724062383174896, 1.4887540340423584, -1.0766382217407227, -0.7541698813438416, 1.0003173351287842, 0.5168557167053223, -0.11273466795682907, -0.5495786070823669, 0.45633548498153687, 0.26536425948143005, -0.25684091448783875, 0.2362929880619049, -0.7247530221939087, 0.8264597654342651, -0.11898829787969589, -0.11679007112979889, -0.06124985218048096, -1.2094804048538208, -0.18218378722667694, -1.726483941078186, -0.1370922178030014, 0.3126285970211029, -1.444705605506897, -1.368755578994751, 1.7874339818954468, 0.26462703943252563, -2.1508615016937256, 1.5026593208312988, 0.8696133494377136, -0.44734033942222595, 0.11470939218997955, -0.00968255940824747, -0.8926493525505066, 0.42432984709739685, 0.04819145053625107, -0.8159494996070862, 0.42826053500175476, 0.788192868232727, 1.4633439779281616, 0.29418566823005676, -1.4859752655029297, -0.4016268849372864, 0.8696155548095703, 1.2485500574111938, 0.29354017972946167, -0.8970584273338318, -0.6161915063858032, -0.7215410470962524, 0.3351750075817108, 0.8209142088890076, -0.07747063040733337, 0.637588620185852, 1.9064438343048096, -0.47456926107406616, -0.02068149857223034, -1.4621702432632446, -0.5171434283256531, -0.37313535809516907, -1.3923683166503906, 1.43264901638031, -2.1321394443511963, -1.0671647787094116, 2.172421455383301, 0.7898533940315247, -0.2157326191663742, -0.4025631248950958, -0.8610366582870483, -0.19041746854782104, 2.244127035140991, 1.2771122455596924], [0.3250466287136078, -0.15259414911270142, 0.5871431827545166, 1.1099181175231934, 0.34033703804016113, -1.3208342790603638, 1.2468417882919312, 0.3472430109977722, -0.39159998297691345, 0.5738029479980469, 0.2276063859462738, -1.3562564849853516, 0.2873029410839081, -1.7221457958221436, 1.0059146881103516, -0.4561605453491211, -0.7946188449859619, 0.5443730354309082, 0.7199884057044983, 1.3358385562896729, -0.08271622657775879, -0.7817784547805786, 1.3103126287460327, -0.6146937012672424, -1.3817873001098633, -0.19000211358070374, 0.7905876636505127, -1.4566478729248047, 0.6072104573249817, 1.7417420148849487, -1.3152844905853271, 0.1441165804862976, 0.08295997977256775, -0.30570217967033386, -0.8660022616386414, 2.603883981704712, 1.1669130325317383, -1.4782768487930298, 0.8940297961235046, -0.5980209708213806, -1.2864338159561157, 0.2032444030046463, -0.1337127983570099, 0.13323357701301575, -1.7238472700119019, 0.8124263882637024, -1.0591822862625122, -1.1962796449661255, 2.5354363918304443, -0.8170068264007568, -0.741606593132019, -1.2504661083221436, 1.8041579723358154, 0.3909158408641815, -1.3044897317886353, -1.344367504119873, -0.28779637813568115, 0.6947163343429565, -2.8000917434692383, -1.00750732421875, -0.4910967946052551, -1.357934594154358, 0.8119722008705139, 1.962623119354248, -0.49544504284858704, 0.5871456861495972, 0.05156215280294418, -0.8339313268661499, 0.3329036235809326, 0.3007547855377197, -1.1922310590744019, -1.1711902618408203, 0.003663767594844103, -1.1943966150283813, 0.991912305355072, -1.013676643371582, -0.27481117844581604, 0.07176998257637024, 0.026305917650461197, -0.5628872513771057, 0.6764388084411621, 0.8176336884498596, 0.3730364441871643, -0.4107167422771454, -1.395161747932434, 1.4852937459945679, -0.0191049724817276, -1.9540808200836182, -1.0858235359191895, 1.4440324306488037, 0.9618337154388428, -0.2909720242023468, 0.5958716869354248, -0.7332944273948669, -1.684049367904663, 1.9421051740646362, 0.05776989832520485, -0.18756352365016937, 0.07893785834312439, -0.22843149304389954, 0.3282323479652405, -0.639202892780304, -0.5862147808074951, 1.314744472503662, 0.15959994494915009, 0.8176035284996033, 0.26171737909317017, 0.6261021494865417, -0.9428250193595886, 1.5910698175430298, 0.7568657994270325, -1.2843539714813232, 0.8466291427612305, 0.7819045186042786, -0.12292401492595673, 0.9773765802383423, -0.1934974044561386, 1.065443754196167, -0.3681338131427765, -0.4912339746952057, -0.918576180934906, -0.07411391288042068, 2.970055341720581, -1.0436301231384277, -2.7924721240997314, -0.18449433147907257, -0.0037414999678730965, -0.06592631340026855], [-0.2932111620903015, -0.13230310380458832, -1.2193635702133179, -1.3454405069351196, -1.2257890701293945, 2.3709757328033447, 0.3356644809246063, -1.189429759979248, -0.29584798216819763, 1.8873223066329956, 0.9437187314033508, 0.28270259499549866, -0.5280676484107971, -0.8859721422195435, 0.4894748032093048, 0.4059118330478668, 0.6437531113624573, -0.6581332087516785, -0.1565907746553421, -1.5926812887191772, -0.15570203959941864, -0.48101699352264404, -0.16970543563365936, 1.6105509996414185, 0.7900223731994629, -1.9880656003952026, 0.6306731104850769, 0.7230502367019653, 1.3087550401687622, -0.14701135456562042, -0.3809206783771515, 0.06876198202371597, -1.3908506631851196, 1.2631219625473022, -0.1414335072040558, -0.9859979748725891, -0.4696592688560486, -0.360839307308197, 0.42708826065063477, -0.9623737931251526, 0.3121051490306854, -0.751729428768158, 0.2196788638830185, -0.5943358540534973, -0.27446699142456055, -0.23504415154457092, 0.28657445311546326, -0.07134414464235306, -0.19723182916641235, 0.2601638436317444, -2.6315996646881104, 0.7129616141319275, -1.9914530515670776, 0.43326708674430847, 1.135970950126648, 1.1711643934249878, 0.43904611468315125, -0.069748654961586, 0.4933331608772278, -0.04286406934261322, 0.5191757678985596, 0.555979311466217, 0.9454874992370605, 0.47990354895591736, -0.03895368054509163, 1.3601902723312378, 1.8080729246139526, -0.9074551463127136, -0.5983127355575562, 1.8859246969223022, 0.1746116280555725, -2.1850719451904297, 0.7992674112319946, -0.8717492818832397, 0.9890255331993103, -0.9735199213027954, 0.4396806061267853, 0.1381695419549942, -1.7027114629745483, 1.4151607751846313, -0.22204124927520752, -0.8125633001327515, -0.20315414667129517, 1.651382327079773, 0.7569078803062439, -0.5839707851409912, 0.7900010943412781, -0.30880194902420044, -0.15175698697566986, 1.8549470901489258, -0.7838394641876221, 0.5533854365348816, -1.194663643836975, 1.734683871269226, -0.9006947875022888, 0.8986705541610718, 0.8383654952049255, 1.5239152908325195, -0.3260570466518402, 1.8827604055404663, -0.3018888533115387, 0.08275879919528961, 0.6578809022903442, -0.3511315584182739, 0.6958437561988831, -1.967590093612671, -1.32901132106781, 1.0819860696792603, 1.9034242630004883, -1.1149834394454956, 0.33824795484542847, 0.8333758115768433, -0.009518750943243504, -1.6837308406829834, 0.512783944606781, 0.4117920696735382, -0.2815984785556793, 0.8109340071678162, -0.22155900299549103, -0.7319427132606506, -0.3733387887477875, 0.5084749460220337, -0.31791478395462036, -1.28507661819458, 0.04410526156425476, -0.7309058308601379, 2.583252429962158, -0.36524802446365356], [0.7197177410125732, 0.4836660325527191, 1.0893683433532715, 0.30089738965034485, 0.42596307396888733, -0.10192219913005829, 0.2484753131866455, 0.3707464635372162, -1.0070209503173828, -0.5778390765190125, -0.7866711616516113, -0.6963123083114624, -1.1708122491836548, 0.26715636253356934, -0.9519907832145691, 0.1763426661491394, -0.46837112307548523, 0.31767114996910095, 0.22249151766300201, 1.150386929512024, -1.0035762786865234, 1.7481459379196167, -1.369645595550537, 1.2642086744308472, 0.9092826843261719, -0.3213339149951935, -0.4382779598236084, 0.4901169240474701, -0.9809914827346802, -0.6808668375015259, 0.37682172656059265, -0.14535507559776306, 0.1830260008573532, 1.4579873085021973, -0.4165448248386383, -0.7183515429496765, -0.3059110939502716, -0.3346541225910187, 0.6962003111839294, 0.887997031211853, 0.299269437789917, -1.325836181640625, 0.2937941551208496, -0.4129321575164795, 2.4995944499969482, -0.7726625204086304, 0.1573077142238617, 0.3386736512184143, 1.2450125217437744, -0.5269407629966736, 0.9302899241447449, 1.235999345779419, 2.3886873722076416, 0.8585778474807739, -1.4773707389831543, 0.8018807172775269, -0.7535738348960876, 0.8731505274772644, -0.1094900444149971, -1.0916327238082886, -2.3301761150360107, -2.043858051300049, 1.6486830711364746, -1.4335967302322388, 1.9288164377212524, -1.545264720916748, -0.06975317746400833, 1.0783405303955078, -2.502031087875366, -1.7071559429168701, 1.3733758926391602, 0.5969521999359131, -0.09006864577531815, -0.7280538082122803, -1.2398616075515747, -0.4592927396297455, -0.6583689451217651, 0.2987477481365204, -0.39202505350112915, -0.6359869837760925, -0.45841220021247864, 1.0749030113220215, -0.9003161787986755, -1.806469440460205, -0.49896299839019775, -0.16045577824115753, -0.6426881551742554, -0.05975012853741646, -1.7657543420791626, -1.0864242315292358, 0.49881574511528015, 1.377691626548767, 0.15132185816764832, -1.3075683116912842, -0.7405800819396973, -0.2259967029094696, -0.5343083143234253, 0.46100717782974243, -0.8748522400856018, -0.5864462852478027, -0.4576623737812042, 0.1324492245912552, 2.0766921043395996, 0.7581313848495483, -0.37252277135849, 0.9589493274688721, 0.0012025191681459546, 0.06765951961278915, -0.4588135778903961, -0.5355366468429565, 1.448952555656433, 0.9065912365913391, 1.4760743379592896, 0.3960496783256531, 0.41777661442756653, -0.9578048586845398, 0.8848563432693481, -0.20224469900131226, -0.0983411893248558, 0.39850279688835144, 1.0851936340332031, -0.39192941784858704, 0.4532259702682495, -0.7192601561546326, -0.46335744857788086, -0.5568006634712219, -0.25053203105926514, -0.41827765107154846], [0.5513116121292114, -0.34311550855636597, 0.454925537109375, -3.009071111679077, 1.81269109249115, 0.3305357098579407, 1.0122709274291992, -2.0695483684539795, -0.05436908081173897, -0.17439870536327362, 0.49956777691841125, -0.19616755843162537, -1.0582078695297241, 0.8359677195549011, -0.3054090142250061, -0.6318855881690979, 0.8699772357940674, -2.1537296772003174, 0.02737155742943287, 2.8649508953094482, 0.3898743987083435, 1.1885615587234497, -1.2384127378463745, 1.6447384357452393, -0.37053316831588745, -1.1396108865737915, 0.6837661862373352, -1.0434626340866089, -1.2132130861282349, 0.6900500655174255, 1.8824853897094727, -0.7778014540672302, -1.579858660697937, -1.8806748390197754, -0.15393628180027008, 1.2737147808074951, -0.07098446041345596, 0.6999719738960266, 1.553226351737976, 0.48685625195503235, 0.5583099126815796, 0.5695085525512695, 1.6365669965744019, -0.09731185436248779, -1.6535868644714355, -0.21871598064899445, -1.7670773267745972, -0.776766836643219, 0.4997209310531616, 1.0433368682861328, 0.8975602984428406, 0.8193880319595337, 0.25258296728134155, -1.7455437183380127, -1.1306023597717285, 0.05762040615081787, 0.4916660785675049, -0.4129939675331116, -0.9472148418426514, -1.4078022241592407, -0.5397443771362305, 0.97725909948349, 0.3028455078601837, -0.8495382070541382, 0.5636820793151855, -1.7689229249954224, -1.2801302671432495, 0.07321271300315857, -0.3476465940475464, -0.7781209349632263, -1.3160886764526367, -0.4422365128993988, -0.902190625667572, 0.37323063611984253, -0.6706060767173767, -1.7524237632751465, 0.8744760751724243, 0.38277924060821533, -0.0008601280278526247, 0.7349449992179871, -0.226084902882576, -0.9631257653236389, -1.0338070392608643, -0.609662652015686, 0.6876091361045837, -1.3135697841644287, -0.9058085680007935, 0.20528042316436768, 1.4675558805465698, 0.7588335871696472, 0.6286939978599548, 0.5612419843673706, -0.0060464125126600266, -1.226691722869873, -1.201554298400879, -0.20750531554222107, -0.8866971135139465, -0.0047117238864302635, -1.1998008489608765, -0.7931584715843201, 0.5708068013191223, 0.14507479965686798, -0.4291701316833496, 0.5953295826911926, 1.7563790082931519, -0.5024926066398621, 1.606939673423767, -0.8828917145729065, 3.259094715118408, 0.5569793581962585, 0.32052335143089294, 0.580987811088562, -0.20067554712295532, 1.644472599029541, 0.6443161368370056, -1.5130339860916138, 1.4682950973510742, -0.09542939811944962, 0.6358928680419922, -0.4155994951725006, -0.8044556379318237, 0.054522011429071426, 0.10347051173448563, 0.6222780346870422, 0.05388018861413002, 0.6205618977546692, -0.6949456930160522, 0.6292359232902527], [0.7165240049362183, 0.8523984551429749, -0.4966700077056885, 0.2879338562488556, 1.6597164869308472, -1.2051962614059448, 1.1268748044967651, -1.6892647743225098, -1.2182918787002563, 0.7297524213790894, -0.14900323748588562, 0.5181775689125061, -0.28073614835739136, 0.7357675433158875, -0.5477038025856018, -0.21678388118743896, 0.8938502669334412, -0.35269322991371155, -0.2306891530752182, 0.360871821641922, 0.5039281845092773, 1.0520738363265991, 0.6863889694213867, -0.004257735796272755, -1.1077244281768799, 0.9043036103248596, 0.5248858332633972, 0.20120996236801147, 0.8375809192657471, -0.06441818922758102, 0.9143885970115662, -1.8221042156219482, 1.0905005931854248, 1.069994330406189, 1.4426109790802002, 0.4684544801712036, -1.0874303579330444, 0.5681594014167786, -1.5036836862564087, -0.9556517601013184, -0.6236855387687683, -1.6614161729812622, -1.1000787019729614, -0.040823791176080704, -0.827062726020813, -0.29748600721359253, 0.16314007341861725, 1.3090929985046387, -0.09196338057518005, -2.187732696533203, -0.7803207635879517, 0.10117601603269577, -0.457685261964798, -2.379783868789673, -1.3808114528656006, 0.9690495133399963, -0.24941663444042206, -0.522703230381012, -1.6645087003707886, 1.2268174886703491, 1.0577908754348755, -0.22838617861270905, -0.8842016458511353, 0.0634426698088646, -0.697750449180603, -1.5774781703948975, -0.36958229541778564, -0.21688610315322876, -1.589718222618103, 1.7764889001846313, -0.9341768622398376, 2.231510877609253, -1.4613384008407593, -0.37756067514419556, -1.5330212116241455, 1.203202247619629, 0.3198357820510864, 0.1426030695438385, 0.91164630651474, -1.374449610710144, -0.059582144021987915, 0.17186763882637024, 1.3811968564987183, -0.0891047939658165, 0.450074166059494, -1.8143895864486694, -0.22521373629570007, 2.0543272495269775, 1.064313530921936, -0.9514193534851074, -0.19199158251285553, 0.16209670901298523, -0.3605574667453766, 0.641353189945221, 1.1477068662643433, 0.28941118717193604, 1.3364605903625488, 0.9069250226020813, 0.8026208281517029, 0.05006938800215721, 0.7496051788330078, 0.5103738307952881, 0.9452590942382812, -0.902428388595581, 0.3812578618526459, -0.43009650707244873, -1.810728669166565, -0.7279929518699646, -1.0188387632369995, 0.9937520623207092, -0.48462820053100586, 1.2616419792175293, -0.627020537853241, -0.21312136948108673, -1.2188541889190674, -0.646919846534729, -0.7938439249992371, 0.7272856831550598, 0.13757453858852386, 0.9166785478591919, 0.5536134243011475, 0.07271024584770203, -1.8426145315170288, 0.530480682849884, -0.8895980715751648, 0.23327277600765228, -0.4727219343185425, 0.4152953028678894], [1.2565666437149048, -0.05021514743566513, -0.24012966454029083, -1.011159062385559, -0.2230914980173111, 1.0443434715270996, -0.24184450507164001, -0.1349116414785385, -0.2206844985485077, -0.7653365135192871, 2.0755600929260254, -0.16438719630241394, -0.2848852872848511, -0.6646531820297241, 0.29758089780807495, -1.7634190320968628, 2.1756391525268555, -1.0801732540130615, -1.2356600761413574, 0.613564670085907, 1.2360284328460693, -0.2126358449459076, -1.468295931816101, -0.8078775405883789, -0.48638594150543213, -1.3196196556091309, 0.4861059784889221, 0.14383843541145325, 0.514939546585083, 1.063650369644165, 0.1577034443616867, 0.02289360761642456, 0.40943655371665955, 1.6138986349105835, -0.0016965785762295127, 0.09283289313316345, -1.1717127561569214, 2.753209352493286, -0.8186193108558655, 0.8169022798538208, 0.09529778361320496, -0.6883029937744141, -0.19318373501300812, -0.4474935531616211, -0.019391098991036415, 0.020417556166648865, -2.455949306488037, -1.8005238771438599, -1.588599681854248, 1.0904711484909058, 2.5525896549224854, -0.279909610748291, 0.8693670630455017, -0.5293795466423035, -1.4129225015640259, 0.7998665571212769, 2.687095880508423, 1.2922784090042114, -1.709909200668335, 0.7718319296836853, -0.1948351263999939, 0.635724663734436, -1.9667601585388184, 0.14886939525604248, -0.9028909802436829, 0.3962942957878113, -0.12202100455760956, 0.509881317615509, 2.099290609359741, 1.8067271709442139, -0.41555607318878174, 1.649251937866211, 1.84408700466156, 0.99382483959198, 1.5198982954025269, 0.29303911328315735, 0.30313464999198914, -0.29119786620140076, 0.5562863349914551, 0.002967487322166562, 0.5062218904495239, 1.2917215824127197, -0.5515186786651611, 0.8573994636535645, -1.2851288318634033, 0.3555039167404175, -0.9204543828964233, 0.6284090876579285, -0.20309677720069885, 0.8138818740844727, 0.04773574694991112, 0.6023393273353577, 0.9030269384384155, 0.2644571363925934, 1.1296027898788452, -1.7357923984527588, 0.8743008375167847, 0.635758101940155, -0.1531345099210739, -0.1530989110469818, 0.6228098273277283, 2.025336265563965, -1.3229855298995972, 0.36886805295944214, -0.048588965088129044, 0.237392395734787, 0.33427560329437256, -1.3321810960769653, 0.46832704544067383, -0.4633623957633972, -0.8915895223617554, -1.7216086387634277, -0.07581961154937744, -0.6168731451034546, -1.2195632457733154, -0.7194733023643494, 1.161586046218872, 0.7292931079864502, -0.6090444326400757, -1.3107913732528687, 0.7227715849876404, 0.3722756505012512, -0.9457376599311829, 0.8359237909317017, 1.8251334428787231, 0.14448963105678558, -0.11872459948062897, 0.4334177076816559], [-0.31264668703079224, 0.8337794542312622, 0.6516939401626587, -0.7117243409156799, 1.1567435264587402, -0.6850190758705139, -1.4613831043243408, 0.04087549075484276, -0.22026407718658447, 1.3702043294906616, 0.6946980357170105, 0.5890924334526062, 0.06871679425239563, -0.02700357511639595, -0.43281829357147217, 0.017327453941106796, -1.1887954473495483, 1.6390918493270874, 0.031095951795578003, -1.7171282768249512, -0.9493089914321899, 1.7760138511657715, -1.517008900642395, 0.019463863223791122, -0.10226259380578995, -0.07935500144958496, -0.5927097797393799, 0.2760958671569824, 2.6789276599884033, -0.3392663896083832, 0.24300211668014526, -1.1838712692260742, 1.722959280014038, 0.7643702030181885, 0.22661332786083221, 1.3993045091629028, 0.024618074297904968, -1.221771478652954, -1.3654361963272095, 0.8027392625808716, 0.003875387366861105, -0.2410724014043808, 1.1382033824920654, -0.5281682014465332, -0.05027325451374054, -0.37216973304748535, 0.4086242616176605, -0.0034849222283810377, 1.4904934167861938, 0.5868389010429382, 0.855829119682312, -1.0201539993286133, -0.29577022790908813, 0.38263410329818726, 0.9871318340301514, 0.8180399537086487, 0.9030272960662842, -1.0891761779785156, 0.19704636931419373, 2.741323232650757, -1.8277417421340942, -1.2326358556747437, -0.7087929844856262, -0.06056235358119011, -0.11728871613740921, -0.4212873876094818, 1.749493956565857, 0.7464087009429932, -0.4118262231349945, -0.011436866596341133, 0.2849067151546478, -0.2906261384487152, 1.5454630851745605, -0.7162644863128662, 0.8162169456481934, -0.7231735587120056, 1.467903971672058, 1.6813441514968872, -0.09141655266284943, 0.882964015007019, 0.17029248178005219, -1.1126775741577148, 0.6793035864830017, 0.8975088596343994, -0.9571933746337891, -1.2776974439620972, 0.9470950961112976, -0.8401296734809875, -0.2711552083492279, 0.37284156680107117, 0.9059924483299255, 0.06480156630277634, -1.366951584815979, 0.3487396240234375, -1.1707111597061157, 0.7950973510742188, 0.4384045898914337, -2.012402296066284, 1.781374216079712, -0.3071358799934387, 1.1258612871170044, 0.3282966613769531, -2.164771556854248, -0.556953489780426, -2.3527822494506836, -0.8186056017875671, 0.9749566316604614, 0.8021014928817749, 0.0654020607471466, -0.13767927885055542, 0.3273935317993164, -0.33037707209587097, -0.4505251944065094, 0.24486880004405975, -0.39430075883865356, 1.6815968751907349, -2.1321234703063965, 0.34406808018684387, -0.007894616574048996, -1.7844061851501465, 1.1516170501708984, -0.17306314408779144, 1.1947169303894043, 0.8434891104698181, -0.6049835085868835, 1.1428759098052979, 0.9625581502914429, -0.13176405429840088], [-1.537039875984192, -1.7796927690505981, -1.1641861200332642, 0.08274312317371368, 0.2948397696018219, -0.4807668626308441, -1.673654317855835, -0.8355907797813416, 0.6650204062461853, 1.0515767335891724, 1.7449814081192017, -1.4548989534378052, -0.690495491027832, 0.41347235441207886, -1.6792359352111816, 0.15717318654060364, -0.8221303820610046, 0.25388118624687195, 0.0820305198431015, 1.2293137311935425, 0.2530927360057831, -0.28191259503364563, -0.17619648575782776, 1.6219605207443237, -0.6884583830833435, -0.1229783296585083, 0.49193090200424194, -2.0146114826202393, -0.42755573987960815, -0.8491193652153015, -0.4409312307834625, 0.2743689715862274, 0.9465736746788025, -0.723851203918457, -3.3563430309295654, -0.3160346448421478, -0.3658709228038788, 0.1354735791683197, -1.4282503128051758, -1.501316785812378, -0.33452755212783813, 0.5797657370567322, 0.6266540288925171, -1.5873554944992065, -0.940228283405304, 0.20358264446258545, -0.1253138780593872, 2.0049850940704346, 0.645361065864563, -0.08941902220249176, 0.5802727341651917, 0.3719709813594818, 0.22333084046840668, -1.7667261362075806, -1.4706857204437256, 0.056662824004888535, 1.413405179977417, 1.7021719217300415, 2.2015554904937744, 1.483197569847107, 0.5192873477935791, 0.8472614884376526, 1.3111941814422607, -0.44606029987335205, -1.0867146253585815, 0.9708684086799622, 1.8436883687973022, -0.8621194958686829, 0.2051762342453003, 1.3086702823638916, 0.24993181228637695, 0.925814688205719, 0.30250588059425354, 0.7440270781517029, 0.8039475083351135, 1.3641908168792725, -0.5043622851371765, -0.11639468371868134, -0.8977652788162231, 0.8112891912460327, 1.8567326068878174, 0.4027596712112427, 0.35342252254486084, 0.42651793360710144, 0.17590674757957458, 0.32906872034072876, -0.9412386417388916, 0.4746551811695099, -0.31834128499031067, -1.547900676727295, 0.21175295114517212, -0.4791608452796936, -0.6537463068962097, -0.6554077863693237, 1.0507023334503174, -0.5374423265457153, 0.5745771527290344, -0.021781517192721367, 1.1139649152755737, -0.63852459192276, -1.1206302642822266, 1.0315420627593994, 1.2468085289001465, 0.5290427803993225, 2.094788074493408, -0.8480331301689148, -0.61360102891922, 0.6408973336219788, -1.2003921270370483, 0.4238766133785248, 0.6498985290527344, -0.02999155968427658, 1.8380022048950195, -0.3190518319606781, -0.16881424188613892, 0.6106098890304565, 2.382627248764038, 1.2276891469955444, 1.482829213142395, -0.7204172015190125, 0.5443226099014282, -0.8115397691726685, -1.4812681674957275, 0.9101402759552002, -0.020704232156276703, -0.16622862219810486, -1.8133796453475952, 1.0504536628723145], [-0.9872454404830933, -1.332004427909851, -0.31829336285591125, 0.6768736243247986, 0.7756835222244263, 0.8964015245437622, -1.5719836950302124, 0.7237978577613831, -0.31758975982666016, 1.2084105014801025, -0.3455631136894226, -0.5536565184593201, 0.08109553158283234, 0.10867836326360703, -0.6544276475906372, 1.3810784816741943, -0.2347581833600998, -0.14644336700439453, -0.8608856201171875, -0.9963887333869934, -0.41591402888298035, 0.23614361882209778, 0.08691921830177307, -1.7269015312194824, -1.5137964487075806, -0.7253245115280151, -0.6688525080680847, 0.3170425593852997, -1.7608611583709717, 0.1656782627105713, -0.23409678041934967, 0.8116233348846436, 0.6052559614181519, 0.03842862695455551, -0.34719231724739075, 1.2132229804992676, -0.8402060866355896, -0.24758513271808624, -2.1051411628723145, 0.569145917892456, 0.9635390639305115, -0.6112846732139587, 1.5126800537109375, -1.9040207862854004, -0.23856276273727417, -0.2933826446533203, 1.7734050750732422, -1.1255849599838257, -0.5038167238235474, 0.967627227306366, -0.5478131175041199, 1.8726866245269775, -1.2036283016204834, -0.42050349712371826, -0.16708973050117493, 1.487804651260376, 2.094770669937134, -0.972628653049469, -1.0179544687271118, 1.2006126642227173, -0.8291254639625549, -0.824219286441803, 0.5292119979858398, 0.3037181794643402, 0.3464793860912323, -1.7640422582626343, 0.7843592166900635, -0.41908615827560425, -0.10935933142900467, 0.43130749464035034, -1.3516777753829956, 1.8519797325134277, -0.3909153342247009, 1.0422271490097046, 1.3147990703582764, -1.175359845161438, 1.4846534729003906, -1.1025948524475098, 0.281324177980423, 2.5804901123046875, -1.7077264785766602, -1.7824984788894653, 1.9567493200302124, 0.17470228672027588, 0.6590259075164795, 0.6856826543807983, -0.10878875851631165, -0.4721776247024536, 1.2135595083236694, 0.2722812592983246, 0.5377423167228699, 0.11426739394664764, -0.6577286720275879, 0.30582207441329956, 0.09119964390993118, 0.4875304400920868, 0.9734408855438232, -0.8888710141181946, 0.712390661239624, 0.4336893558502197, 0.1415388584136963, -1.0118249654769897, -0.47493812441825867, 0.13280075788497925, -0.8712392449378967, 0.39557045698165894, -0.616009533405304, 0.12976515293121338, -1.7311981916427612, -0.8485095500946045, -3.28653621673584, 1.7842885255813599, -0.9401823878288269, -0.17323215305805206, -1.0001260042190552, -1.101996660232544, 0.4120391607284546, -0.3610392212867737, -1.0349282026290894, 0.8904671669006348, -1.8013181686401367, -0.2947675585746765, 0.4832351803779602, -0.9799253940582275, -0.3902130722999573, 0.21949949860572815, 0.7674292325973511, 1.4867045879364014], [0.7745168805122375, -0.9023313522338867, -0.7837731838226318, -0.9074510335922241, 0.04484066367149353, 0.1048363521695137, 0.993967592716217, -2.028278350830078, -0.19499875605106354, -0.5683147311210632, -1.2677452564239502, -0.22721543908119202, 1.0517417192459106, 0.08460526913404465, -0.7904108166694641, -1.2932114601135254, -0.6627331972122192, -0.3480501174926758, -1.6239616870880127, -0.5902851819992065, 1.7002193927764893, 0.9722424149513245, 0.3923276960849762, 0.6416928768157959, -0.48159661889076233, 1.890053391456604, -2.1920595169067383, 0.18018649518489838, 0.32461461424827576, -0.9412063360214233, 0.7128437161445618, -0.42075115442276, 0.5028529763221741, -0.46648266911506653, 0.0019571532029658556, -0.5485085844993591, -0.44764620065689087, 1.038842797279358, -1.6598187685012817, -1.1935505867004395, 1.18582284450531, -1.4007092714309692, -0.016054103150963783, 0.32576826214790344, -0.3860907554626465, -1.2800099849700928, -0.7116096019744873, -0.22796596586704254, 0.7612624168395996, -0.7820675373077393, -1.111521601676941, 0.5487907528877258, -0.7519760727882385, -1.9823260307312012, 0.1254272609949112, 0.40483468770980835, 1.4349119663238525, -0.4380383789539337, 1.1394398212432861, -0.07154589891433716, 1.825844407081604, 0.30546507239341736, -1.8021519184112549, 2.11336088180542, 0.05381692200899124, -0.9818361401557922, -1.1495599746704102, -1.340733528137207, 2.001858949661255, -3.363964319229126, 0.5306057929992676, 0.43653491139411926, -0.7117525935173035, 0.7746977806091309, 0.3403245210647583, 2.078564405441284, -0.3241167962551117, 0.6789897084236145, 0.060454998165369034, -1.5316500663757324, -1.1685960292816162, -0.37978506088256836, 0.4148605763912201, -0.5852217078208923, -0.4136850833892822, -0.026823366060853004, -0.4151172339916229, 0.16045726835727692, 0.701420247554779, 0.4770796597003937, 0.8798915147781372, -0.879492998123169, 0.5971494317054749, 0.6481996774673462, -1.21241295337677, -0.5161004066467285, -0.2517854571342468, -0.9402661323547363, 1.9986951351165771, -0.9868354201316833, 0.2865910530090332, -0.5017353892326355, 0.08744952082633972, 1.837022066116333, -0.6963350176811218, 0.43549296259880066, -0.1892227828502655, 1.3386752605438232, -0.015057873912155628, 0.5604950785636902, 0.4000062346458435, 1.0603644847869873, 2.003031015396118, 1.0390751361846924, -1.2271943092346191, -0.8993044495582581, -0.2672240734100342, -0.49252450466156006, 0.30727618932724, 1.5829704999923706, 0.6931464672088623, 0.04256065934896469, -1.7651060819625854, -0.3429046869277954, -0.5159236192703247, -0.23920132219791412, -2.3110461235046387, 0.5746459364891052], [-0.04790869355201721, -0.14990967512130737, -0.36104318499565125, 1.0086853504180908, 0.4109744727611542, 0.1463862806558609, 1.0914323329925537, 0.7946382761001587, -1.1843739748001099, 0.45665308833122253, 1.5811610221862793, 0.8419339656829834, 0.743827760219574, -0.6922826766967773, 0.9269388914108276, -1.925373911857605, -0.29327720403671265, -0.07031508535146713, 0.36446937918663025, -0.040046725422143936, -0.05824674293398857, -0.8137903809547424, 0.6605480313301086, 0.5154947638511658, -0.6465844511985779, 0.27764660120010376, 0.38279691338539124, -0.7475467324256897, -0.5801173448562622, -1.413084626197815, -9.194903213938233e-06, -0.5182446837425232, 0.250660240650177, -0.5914282202720642, -1.3350307941436768, -0.3012944757938385, -0.800622820854187, -1.3849101066589355, -0.5066794753074646, -0.6877947449684143, 1.4143767356872559, -1.4819536209106445, 0.6278700232505798, 1.413347840309143, 0.9044667482376099, -0.9041612148284912, -1.2782063484191895, -1.1083076000213623, -0.6341636776924133, -2.4279966354370117, -0.3992476761341095, -0.3933698832988739, -0.3930785059928894, -1.3128994703292847, -0.031703148037195206, -1.2886428833007812, 2.3115427494049072, -0.6119480133056641, -0.35367098450660706, -0.4775497615337372, -1.20103120803833, 0.6835257411003113, 1.1498256921768188, 0.14349038898944855, 2.0116231441497803, 1.4916938543319702, 0.2777058780193329, 0.42550399899482727, 0.4425378143787384, -1.2121912240982056, -0.7114737629890442, -0.5393263101577759, -0.5966299772262573, 1.4133515357971191, 0.8567023873329163, -1.0193934440612793, 1.5437263250350952, 0.4025858938694, -1.7832443714141846, 3.153712272644043, -0.4676871597766876, -0.8576942682266235, 1.2243188619613647, -0.27955347299575806, 0.05743815377354622, -1.5057333707809448, -0.8681856989860535, -1.272728681564331, -0.38270217180252075, 1.0369179248809814, 0.9413869976997375, 0.5893739461898804, 0.8905807733535767, -1.3456693887710571, -0.8047630786895752, 0.21136000752449036, -1.2180367708206177, -0.47501248121261597, 0.5933789610862732, 0.3587406277656555, -1.1848551034927368, -0.9279890060424805, 0.4000551104545593, -0.2687928378582001, -0.96356201171875, 0.7888712882995605, -0.7970948219299316, -0.030697694048285484, -0.744381308555603, -0.6090039610862732, 1.8347686529159546, 1.7568111419677734, -0.7957988977432251, 1.1292603015899658, -0.2839817702770233, -1.4591258764266968, 0.5675930380821228, 0.8020975589752197, 0.2863004207611084, 0.3687734007835388, 0.9693037867546082, 0.2772623896598816, 1.7656162977218628, 0.8158847689628601, -0.6088916063308716, -0.16843575239181519, 0.819179892539978, -0.7877398729324341], [0.584380567073822, 1.714127779006958, -0.5808266401290894, -0.6616004109382629, 0.3029765784740448, 0.4232144057750702, -1.1938360929489136, -0.5747410655021667, 0.23183177411556244, -0.7215277552604675, 0.7912909388542175, 0.05547093227505684, -0.7788790464401245, 1.8493337631225586, 1.9151066541671753, -0.6882320642471313, 1.835187315940857, 0.005115824285894632, 1.735782504081726, -0.9862898588180542, -0.34372609853744507, 0.6805189251899719, 0.8289587497711182, -0.18105991184711456, 0.3130098879337311, 0.343403160572052, 0.5181646943092346, -0.1007615327835083, -1.1044948101043701, -1.9688746929168701, 0.6588712930679321, -0.05249454081058502, 0.6116201877593994, -0.4340014159679413, 0.11977685242891312, -2.286587953567505, 0.11436706781387329, 0.6768357753753662, -0.09072252362966537, -0.6833965182304382, 1.286714792251587, -0.10638095438480377, 0.02183588221669197, 0.021533340215682983, -0.778021514415741, -0.07661886513233185, -0.7388319373130798, -1.5373151302337646, -0.1513594686985016, -1.2710703611373901, 0.5709294676780701, 1.389743447303772, -0.05896029621362686, -0.7673498392105103, 0.7589558959007263, -2.988382339477539, 1.0967717170715332, 0.004162818659096956, -0.5198491811752319, -0.01684422977268696, -1.1659412384033203, -0.759528636932373, 1.9143998622894287, 1.2875993251800537, 0.12403065711259842, 0.5841834545135498, -1.3783724308013916, 0.3159996569156647, -0.3741213381290436, -0.9226585626602173, 0.24290569126605988, -0.6555785536766052, 0.9973818063735962, -0.45477306842803955, -0.9631419777870178, 1.32426118850708, -0.6965678930282593, 0.6272339820861816, 0.17410723865032196, -0.1602872908115387, -1.5210373401641846, 1.166191577911377, -0.9123797416687012, 0.0809539183974266, 2.004227876663208, 0.15407618880271912, 0.008350054733455181, 0.8121008276939392, -0.23110103607177734, -1.3781489133834839, 0.8497691750526428, -1.550412893295288, -0.2939148247241974, 0.34000730514526367, -0.3887307047843933, -0.30940842628479004, 0.8200430870056152, -1.7739427089691162, 0.46119800209999084, -0.9169033169746399, -0.4008443057537079, -0.79215008020401, -0.974338173866272, 1.0658305883407593, 0.9544181227684021, -0.39208394289016724, 2.373188018798828, 0.42025673389434814, 0.19765500724315643, 0.03575199842453003, 1.0404744148254395, -0.6643825769424438, -1.5470739603042603, 0.17340654134750366, -0.28193625807762146, -1.860477328300476, 0.007484983187168837, 0.9040926098823547, -0.8680633306503296, 0.09911425411701202, -1.308470606803894, 1.0427203178405762, -0.2556806206703186, 1.4108701944351196, -1.1794699430465698, 0.10463675856590271, 0.401540070772171, -0.7008029818534851], [0.37142759561538696, -1.402951717376709, 0.7086707949638367, -0.23358146846294403, -0.9794336557388306, -1.8911762237548828, -0.027329372242093086, -0.2971726357936859, -0.5520328283309937, -0.9090096950531006, 1.8395100831985474, 0.4737985134124756, 0.7070008516311646, -0.6059086918830872, -1.679520606994629, 0.8493273258209229, -0.5618044137954712, -0.5304853916168213, -1.3938418626785278, -1.6207764148712158, -2.014664649963379, -1.0852240324020386, 2.2010655403137207, -0.13446582853794098, 0.07975992560386658, -0.27175331115722656, -0.2587456703186035, 2.1969854831695557, -2.7330713272094727, -0.5170018672943115, -1.6152656078338623, -0.6191781163215637, -0.1801712065935135, 1.0807908773422241, 0.913072407245636, -1.616672158241272, 1.471084713935852, -2.2103614807128906, 1.3813509941101074, 1.644966959953308, 0.8721230626106262, 1.4651472568511963, -0.5170511603355408, 0.14949384331703186, -0.017072072252631187, 1.2287429571151733, -0.2744937837123871, 0.635829508304596, 0.9103612899780273, -0.33052971959114075, -0.6626389026641846, -0.4141973853111267, -0.1267838478088379, -0.40075668692588806, -1.5414891242980957, 0.6396909356117249, 0.17342044413089752, -0.6266544461250305, 0.34785833954811096, 0.6741167306900024, 0.6332198977470398, 0.09377934038639069, 1.746209979057312, -1.6784807443618774, -0.33659231662750244, -0.12077819555997849, -1.5765846967697144, -0.29140153527259827, -0.025003118440508842, -0.1271185427904129, -1.2803090810775757, 0.40311846137046814, -0.13720031082630157, -0.2601802945137024, -0.9644997715950012, -0.26037275791168213, 0.13730403780937195, -1.441907525062561, 1.7270560264587402, 0.03630264848470688, 0.5047523975372314, -0.6018571257591248, 0.7151466012001038, -0.5812039375305176, 0.8755007386207581, 0.49754709005355835, 0.724782407283783, 0.5818828344345093, -0.17856532335281372, 0.5634610652923584, 0.9617671370506287, 1.2912623882293701, 0.5379529595375061, 0.4238773286342621, 0.6832599639892578, -1.0873479843139648, -0.12958811223506927, 0.3922120928764343, -1.1514039039611816, -0.08084506541490555, -1.112207293510437, -0.9333618879318237, -0.7411324977874756, -0.660373866558075, 0.17824237048625946, 0.4298473596572876, -0.625575840473175, -0.5790404081344604, 0.384259968996048, 0.022533204406499863, 0.06769521534442902, 0.9329521059989929, 0.38429221510887146, 1.1478979587554932, 0.6175681948661804, -1.804099440574646, 0.18078981339931488, 0.06875511258840561, 0.2761569023132324, -1.4150331020355225, -1.3936967849731445, 2.248922824859619, 4.358604907989502, -0.23570095002651215, 1.5814933776855469, -0.823512613773346, 0.4220164716243744, 1.0350184440612793], [-0.047910597175359726, -0.1393030434846878, 0.054913874715566635, 1.758438229560852, 0.6383301019668579, -0.7584896683692932, 1.2260525226593018, -1.2734997272491455, 0.4472614824771881, 1.4028748273849487, -0.6464200615882874, 0.169411301612854, -0.9020417332649231, -0.1666932851076126, 0.2422104924917221, 0.3314616084098816, -1.0955970287322998, 1.2972856760025024, 0.009398405440151691, 1.51398503780365, -0.7801210880279541, 0.8113806247711182, -1.0747557878494263, 1.0939078330993652, 1.0788826942443848, 0.6662203669548035, 1.5732250213623047, -0.2813795506954193, 0.44873368740081787, 0.445791631937027, -0.6618408560752869, -0.33187735080718994, -0.26654890179634094, -0.13627228140830994, 0.8808578848838806, -0.3244016766548157, -0.21212051808834076, -0.6115747094154358, 1.4917651414871216, -0.7600032091140747, 0.19628283381462097, 1.0864136219024658, 0.10372629016637802, -0.3908657133579254, 0.5759220123291016, 1.2870007753372192, 1.3339016437530518, -0.19088804721832275, 1.738630771636963, 1.3258262872695923, -0.3083176612854004, -0.4976717531681061, -0.1827106475830078, 1.0558674335479736, 0.28500446677207947, 0.49531692266464233, -0.5154137015342712, -1.25505530834198, 1.5361937284469604, -0.7436500787734985, 0.731525182723999, -2.064697504043579, -0.10298340022563934, -1.2472832202911377, 1.3608779907226562, 2.212216377258301, 0.471732497215271, 0.45671606063842773, -1.4924805164337158, -0.4603644907474518, -0.2151189148426056, 0.8933348059654236, 1.7212263345718384, 1.2845731973648071, -0.15953011810779572, 1.0322861671447754, 1.4651908874511719, -1.0916228294372559, 0.7146594524383545, 0.38602590560913086, -0.4392760992050171, 0.4136350750923157, 0.057581834495067596, 0.017004765570163727, 0.18835189938545227, 0.19570426642894745, -1.4730565547943115, 2.614718437194824, 1.014366865158081, -0.2531515657901764, -1.433345913887024, 0.873544454574585, -1.6838988065719604, 0.030908750370144844, -0.7066220641136169, -0.025289997458457947, -1.0779407024383545, -0.7191352844238281, -0.6663898825645447, 0.3611823320388794, -1.1544352769851685, -0.7124518752098083, -0.9440277218818665, -0.10561036318540573, -0.7616608738899231, 1.1701279878616333, 1.3875001668930054, 0.5056098103523254, -0.784404456615448, 0.10508893430233002, -0.36758843064308167, -1.421043038368225, -1.2399823665618896, 0.1802535504102707, 0.9459788203239441, 0.9518548250198364, -0.8940567374229431, 1.17146897315979, -0.811478316783905, 0.3368326723575592, 0.5490419268608093, 0.9618152976036072, -1.9054031372070312, -0.6247465014457703, 0.22914084792137146, -0.486115425825119, -0.24334973096847534, 0.2850172817707062], [-0.0665978342294693, 0.5774744153022766, 1.6174507141113281, -0.2825172245502472, -1.8700740337371826, -0.6970918774604797, -0.395900696516037, 1.2494685649871826, -0.5729560852050781, -0.7319042682647705, 0.6554646492004395, 0.0033705884125083685, -0.7323703169822693, 0.1743689328432083, -0.6563730239868164, -0.4941580891609192, -1.422794222831726, 1.2093371152877808, 0.41433367133140564, 0.5367137789726257, 0.7904012799263, 0.9505255222320557, -0.42106911540031433, -0.943786084651947, -0.15301190316677094, 0.1850252002477646, 0.10606583952903748, -0.6224386096000671, 0.23419328033924103, -2.4404194355010986, -0.5793574452400208, -0.26594123244285583, 1.4924565553665161, -0.09462476521730423, -1.295303463935852, 1.1840102672576904, 0.014728960581123829, -0.5175852179527283, 0.1488344520330429, 1.054891586303711, 0.48088932037353516, -0.0030144795309752226, 0.9749372601509094, 0.7114549279212952, -0.6494886875152588, 2.379976749420166, -0.45074930787086487, 0.006954123266041279, 0.7897757291793823, -0.33524587750434875, -0.10209210962057114, -0.9120083451271057, -0.7821508646011353, -2.0767667293548584, -0.10480006784200668, -0.3692735731601715, -0.1962941735982895, -0.8569450974464417, -0.9660685658454895, 0.9328314065933228, -0.8161674737930298, -1.241485834121704, -0.3049122095108032, 1.780051827430725, 0.04242604970932007, -0.8727832436561584, -0.38022086024284363, 1.3960939645767212, 0.6599118709564209, -0.7910925149917603, 1.1601473093032837, -1.198646068572998, 0.8994495272636414, 0.9600647687911987, -0.8187552690505981, 0.03882750868797302, 1.1049256324768066, 0.292883962392807, -1.237494707107544, -0.7561646699905396, 1.7331902980804443, -1.6042519807815552, 0.33373022079467773, 0.03678373992443085, 0.003540425095707178, -0.02353716269135475, -0.011711055412888527, -1.3504198789596558, 0.37839046120643616, -1.950056791305542, -0.12032521516084671, 1.2486064434051514, -0.9795184135437012, -0.20787182450294495, 0.10507408529520035, 0.9175941944122314, 0.38297683000564575, 0.1979532688856125, 0.18396106362342834, -0.04107293114066124, 0.10722744464874268, 0.20192226767539978, -0.6168932318687439, 0.32057905197143555, 2.240744113922119, -0.14122121036052704, -2.2184550762176514, 0.4194377362728119, -0.6020340919494629, 1.8234143257141113, -0.1335594207048416, 0.30089595913887024, -1.3471194505691528, -1.7394864559173584, -1.7725125551223755, 0.17948825657367706, -0.8274834156036377, 0.44727393984794617, -0.7457470297813416, -0.014067397452890873, -0.5051948428153992, -0.16149958968162537, -0.015246212482452393, 0.5492268800735474, -0.4191163182258606, 1.40302312374115, 0.4081328213214874, 0.9852411150932312], [-0.15183159708976746, -0.16706188023090363, -0.36575403809547424, -0.813857913017273, -0.41762542724609375, 0.5077590346336365, -0.18336908519268036, 0.650534987449646, -0.9972010850906372, 0.5439095497131348, 2.321788787841797, 0.6608576774597168, -0.5477578043937683, 0.03174911066889763, 0.01787007786333561, -0.03283609449863434, -1.1909259557724, 0.37187063694000244, -0.705643892288208, -0.49023905396461487, -0.769232988357544, -0.9869892001152039, 0.045447323471307755, 0.8051121234893799, 0.9339103698730469, 2.2181034088134766, 0.7102706432342529, -0.2279803305864334, -0.8742005825042725, -0.04599789157509804, 0.9695356488227844, 0.12678943574428558, 0.9377791285514832, 1.0718727111816406, -1.5964562892913818, -0.4105127453804016, 0.45795977115631104, 0.38448837399482727, 1.005271315574646, 0.5542900562286377, -0.7653142809867859, 1.3978465795516968, 0.2495957911014557, -0.2749869227409363, -0.19364352524280548, -0.5802819728851318, 1.771265983581543, -1.0933603048324585, 0.5816828608512878, -0.16719897091388702, -1.4775919914245605, -0.8762372732162476, 0.47042134404182434, 0.008640236221253872, -0.04630802571773529, 0.15472103655338287, 1.32123601436615, 0.31997957825660706, 0.4487183392047882, -0.1076895147562027, 0.6448518633842468, 0.10805486142635345, -1.173850655555725, 1.34564208984375, -0.08554978668689728, -2.5752601623535156, -0.5050281286239624, 0.1047452762722969, 0.5564173460006714, 1.2255849838256836, 0.3391115665435791, 0.3916318118572235, -0.7461783289909363, -1.4037253856658936, 0.14740155637264252, -0.09113136678934097, -0.823282778263092, 0.9682642221450806, 0.010856913402676582, 0.4527416527271271, 0.2840088903903961, 1.9214202165603638, -0.5658465623855591, -0.021328147500753403, 0.37143880128860474, 1.1274229288101196, -0.86142897605896, 0.6759090423583984, 0.2264501303434372, 0.46319645643234253, -1.049562692642212, 0.8611374497413635, 1.2667226791381836, 2.0867457389831543, 0.268170565366745, 0.9731405973434448, 1.132347583770752, 0.82026606798172, 0.48806577920913696, -2.041788101196289, -0.3364666700363159, -1.3525559902191162, 0.6552087664604187, 0.556949257850647, -0.5957756638526917, 1.1564074754714966, 1.8935662508010864, -2.073348045349121, -0.8099920153617859, -2.1011226177215576, -0.6283133625984192, -2.3193020820617676, 1.7438063621520996, -1.9446296691894531, 0.7839383482933044, 0.9304218292236328, 2.535193681716919, -1.8397020101547241, 0.42669224739074707, -0.1471589207649231, 0.6127016544342041, -2.9699037075042725, -0.5833072066307068, -0.3305637836456299, -2.1238298416137695, 0.6725002527236938, -0.07859666645526886, -0.7361447811126709], [0.6794998049736023, 0.022165250033140182, 1.9983729124069214, -0.016270671039819717, 1.3445992469787598, -0.006795375607907772, 1.2963377237319946, 0.5573975443840027, 1.9467600584030151, 0.00995697919279337, -1.685526967048645, -0.1866704523563385, 1.0871446132659912, 1.0820629596710205, 1.083162784576416, -0.10938672721385956, 0.24177761375904083, 0.3949261009693146, 0.9519864320755005, 0.4348940849304199, 0.0918232724070549, 0.04351839795708656, -2.2091569900512695, -1.3321903944015503, -1.1192853450775146, 2.509307384490967, 0.016771186143159866, -0.03759114071726799, -0.08247633278369904, 0.08818496763706207, 0.6877713203430176, 1.4805632829666138, 1.1018741130828857, -0.9641140699386597, -1.0143227577209473, 0.5332155823707581, 1.9701824188232422, 0.3487509489059448, -0.8085014820098877, 0.3460134267807007, -0.13089419901371002, 0.6210657358169556, 0.6171274781227112, -0.27813011407852173, -1.0881807804107666, 1.4628416299819946, -0.3792543113231659, 0.5820035934448242, -1.1965789794921875, 0.7050354480743408, 0.06833545863628387, 0.4639655351638794, 1.226170539855957, 1.8802998065948486, 0.7803182005882263, 1.1224902868270874, 1.2576671838760376, 0.4358915388584137, -0.2005111277103424, -1.6030213832855225, 0.672620952129364, -0.09801211208105087, 0.7690466642379761, -0.4741082191467285, -1.390460729598999, -2.3296420574188232, 0.750062882900238, -1.9008721113204956, 0.15450012683868408, 1.2368346452713013, 0.043299317359924316, 1.161615252494812, 0.46120065450668335, -0.27856358885765076, -0.14605270326137543, 0.4435986578464508, -0.6659945845603943, 1.314245343208313, 0.09969472140073776, -2.2940900325775146, -0.3692382276058197, -0.08424046635627747, 0.553219735622406, 0.19113479554653168, -1.2296347618103027, -1.7395744323730469, -0.4984363615512848, -0.2217453122138977, 0.08748384565114975, -0.656165361404419, -0.3091678321361542, -1.219706654548645, 2.149620771408081, -0.6488037705421448, -0.7340041995048523, 0.18828709423542023, -0.8647592067718506, 0.7906521558761597, -0.811296284198761, 0.4728144407272339, 0.13225547969341278, 0.5449135303497314, 0.6481083631515503, -1.0212806463241577, -0.8784512281417847, 0.6519984006881714, -0.7346042394638062, -0.35908570885658264, -2.111783742904663, 0.6790763139724731, -1.4615757465362549, -0.4378073513507843, 0.35560137033462524, -0.5334811210632324, -0.02495720610022545, -1.5698981285095215, -0.013207743875682354, -0.6478542685508728, 0.16250157356262207, 0.41999316215515137, 1.6951730251312256, -0.5664528012275696, 2.274369478225708, 0.5554568767547607, -0.03947976604104042, -1.1551786661148071, 0.5683507919311523, 1.7537177801132202], [1.4277889728546143, -0.5077787041664124, 0.8002554178237915, -1.79495370388031, 1.5349373817443848, -0.7545810341835022, -0.8134079575538635, 1.4230668544769287, 1.5107998847961426, -1.2717067003250122, -1.0656572580337524, 0.8447075486183167, 0.07028065621852875, 1.2625774145126343, -1.4179013967514038, -1.7798969745635986, -1.2767452001571655, -0.4888570308685303, -0.8397715091705322, -1.6185520887374878, -1.0457857847213745, 1.134805679321289, 0.5267515182495117, 0.5681887865066528, 0.897834062576294, -0.8110030889511108, -0.22537213563919067, 1.214949369430542, 0.2802242636680603, 0.10825338959693909, -3.2027018070220947, -1.2408086061477661, -0.5212958455085754, -0.22428274154663086, -0.9261462092399597, -0.20188766717910767, 0.05240604653954506, -1.7948830127716064, 0.619334876537323, 0.5702977180480957, 1.6501483917236328, -2.069802761077881, 1.5997387170791626, 1.1321444511413574, -0.0061778961680829525, -0.9499396085739136, -0.7502478361129761, 0.9850296378135681, 0.7708854079246521, 0.13739249110221863, 1.733168363571167, -0.9347820281982422, 0.5757092237472534, -0.023166486993432045, 0.2689339816570282, -1.379711389541626, -0.07124168425798416, -0.8855209946632385, 0.7249020934104919, 0.21713747084140778, -1.01665461063385, -0.30404365062713623, 1.7901438474655151, 0.3239295482635498, 1.5807427167892456, 0.16414473950862885, 0.05537023767828941, -0.47874054312705994, -0.1309138834476471, 1.0794906616210938, -3.047710657119751, 1.851029634475708, -1.655150055885315, -1.6001734733581543, -0.9985303282737732, -0.1667707860469818, 1.4158899784088135, -1.1427414417266846, 1.0288389921188354, -0.3201514184474945, 1.565041184425354, -0.669044017791748, -0.29232436418533325, 0.274644136428833, 0.6657426953315735, 0.7816619873046875, -0.8288454413414001, -0.22105242311954498, -0.9952560663223267, 1.2569853067398071, 1.922616720199585, -0.19988591969013214, 0.0738605409860611, -1.0520528554916382, 1.5278041362762451, 1.4715322256088257, -0.42255571484565735, -0.5377863645553589, -0.18464595079421997, -0.7561538219451904, 0.4606516659259796, -0.1485259085893631, 0.5927557349205017, -0.3545779585838318, 1.4825801849365234, -0.16783633828163147, -0.359992116689682, 0.6829591393470764, -0.6223347187042236, -0.08851414173841476, 1.5245157480239868, 0.17096726596355438, 0.0858621746301651, 0.42434850335121155, 0.334597647190094, -0.37430456280708313, 1.5342234373092651, -0.7231512665748596, -0.8078476190567017, 1.4873846769332886, 0.3477611243724823, 0.7868952751159668, -0.2714673578739166, -0.048451684415340424, -1.9503469467163086, 0.6328138709068298, 0.6094080805778503, -1.758577585220337], [-0.8413662314414978, 0.14218489825725555, 0.09139638394117355, -0.4023362100124359, 0.27367743849754333, -0.9608399868011475, -0.33905228972435, -1.1466761827468872, 0.7390438914299011, -0.9431633949279785, 0.9655191898345947, 1.0279110670089722, -0.09924530237913132, 0.8274127840995789, -0.8696795701980591, 1.1360807418823242, -0.370668888092041, -1.2434724569320679, 0.04063122346997261, 1.7501180171966553, 0.44425204396247864, 0.5171027779579163, 0.10495748370885849, -1.3535064458847046, -0.6474796533584595, 1.3862189054489136, 1.4134467840194702, 0.3655407428741455, 1.157476782798767, -1.1865572929382324, -1.0171012878417969, -0.1753108948469162, -1.4897027015686035, -0.07828713208436966, 0.4671855866909027, 0.7588992714881897, 0.08910617232322693, 0.5883067846298218, -1.5189094543457031, -0.7937068343162537, -1.199654459953308, 1.309449553489685, 0.9991366863250732, 0.1678904891014099, 0.3677726089954376, -0.6340608596801758, -0.30520662665367126, 0.3435458242893219, 1.2312486171722412, -0.16396769881248474, 0.24657638370990753, -1.1412065029144287, -0.2388608753681183, 0.049774594604969025, 0.7950325608253479, -1.1128543615341187, 0.5820239782333374, 1.0237621068954468, -1.3649959564208984, 0.0840863585472107, -0.17730282247066498, -0.6381618976593018, 0.05722283199429512, -0.5898563861846924, 0.3671983778476715, 0.7055893540382385, 0.5365012288093567, -0.36559173464775085, -0.29200613498687744, 0.33765020966529846, -0.566688060760498, -0.3845818042755127, -0.9701922535896301, 0.05194608122110367, -1.0157654285430908, 0.8879066109657288, 0.8834234476089478, -0.45682263374328613, -0.8718293905258179, 0.09452907741069794, -0.28976115584373474, 0.5935054421424866, -0.8426036834716797, 0.32462525367736816, -0.3573553264141083, 0.5287622213363647, -0.5306700468063354, -0.6688032746315002, 0.4372193217277527, -1.289649248123169, 0.042190324515104294, 0.2621501386165619, -0.14636220037937164, -0.7251753211021423, 1.4819201231002808, -0.7888250350952148, -0.06557520478963852, -1.0719163417816162, 0.7068378329277039, -0.25597214698791504, -0.6675506830215454, -0.22071997821331024, -0.8154217600822449, -1.0417660474777222, 0.6458847522735596, -0.6191310882568359, 1.28413724899292, -0.5260056853294373, 0.3652370870113373, -1.108115315437317, 0.1411089450120926, -0.03738950192928314, -1.81374990940094, 0.9755957126617432, -0.13007943332195282, -0.4232534170150757, -0.1007009893655777, -0.2434919774532318, -0.09113409370183945, 0.9342483282089233, 1.0660958290100098, 1.3439266681671143, -0.6533997058868408, 0.4666486084461212, -0.024484509602189064, -0.29121142625808716, -0.1517532914876938, -0.5599823594093323], [-1.2374067306518555, 0.1319171041250229, 0.7481585741043091, 0.366456001996994, 0.5849223732948303, 0.35707908868789673, -1.0044788122177124, -0.1925763040781021, 1.60175621509552, -1.604398488998413, -0.8917941451072693, -1.3288952112197876, 0.7053176164627075, 0.0986725315451622, 0.3154184818267822, -1.3797446489334106, 0.6746628284454346, -1.1057026386260986, -0.08152831345796585, 0.11902347952127457, -0.7683528661727905, 1.6794012784957886, -0.06797205656766891, 1.3472681045532227, 0.03827684745192528, 1.0302636623382568, -0.8181673288345337, 1.4843844175338745, -0.5521268844604492, 0.2618122696876526, 1.3958796262741089, 0.5884624719619751, 0.6099175810813904, -0.06572728604078293, -2.1050267219543457, 0.98062664270401, -1.9217005968093872, 0.7752792835235596, -1.1923433542251587, 0.03226356580853462, -1.1799505949020386, 0.03117888607084751, 0.7462127804756165, 0.930776059627533, 0.624301552772522, 1.2160123586654663, -1.329197645187378, 0.214454784989357, -0.2977760136127472, 0.4112553894519806, 0.8893689513206482, -0.6401824355125427, -2.796243190765381, -1.2551714181900024, -1.5263313055038452, -0.17816969752311707, -0.8939415216445923, -0.8740542531013489, 0.3726027309894562, -0.4716539978981018, -0.5185637474060059, 1.0864803791046143, 0.496930330991745, 2.1910343170166016, 0.8954796195030212, 0.44169363379478455, -0.5645033717155457, -1.3263754844665527, -0.07012781500816345, 0.15996012091636658, -1.4239617586135864, 0.06409171968698502, 1.2471643686294556, -0.5211272835731506, -2.4480090141296387, 2.0029385089874268, -2.525946855545044, 0.542710542678833, 0.519729495048523, -1.8121840953826904, 0.22227932512760162, 0.47346264123916626, -0.08002375811338425, -0.037274740636348724, -0.6907032132148743, 0.9445055723190308, 0.286235511302948, -0.3308540880680084, 1.3979052305221558, -0.6096917986869812, -0.38148894906044006, 0.5375292301177979, -0.4597528874874115, -2.465935230255127, -0.49489110708236694, 1.390336275100708, 0.5878660082817078, -1.5343939065933228, 0.8751912713050842, -0.8100041747093201, -0.7356027364730835, -0.8278424739837646, 0.8982670307159424, 0.7954531311988831, -0.6614052653312683, -0.46670299768447876, 0.26910102367401123, -1.0850597620010376, -0.39695659279823303, -0.20707261562347412, -1.2028508186340332, 1.6049163341522217, -0.5739596486091614, 0.5852039456367493, 0.14400391280651093, 1.8218978643417358, -1.0872536897659302, -0.012571997940540314, -0.08832374960184097, 0.8405089974403381, 0.9367384314537048, -2.284715175628662, 1.181793451309204, 0.04248080402612686, -0.7200721502304077, -0.6790975332260132, 1.5437830686569214, 2.278428077697754], [1.806216835975647, -1.0932880640029907, 0.961868405342102, -0.12899833917617798, 0.6462434530258179, -0.5202961564064026, 1.9361563920974731, 0.8113605380058289, -3.276655673980713, 0.31526586413383484, -0.6711584329605103, 0.9026604294776917, -0.48827865719795227, 1.227539300918579, 0.08346289396286011, -0.5429620742797852, -0.8584861159324646, 0.3690301775932312, 0.06272808462381363, -0.516524076461792, -0.14042212069034576, 2.790799140930176, 0.4599955081939697, 0.5305315256118774, 2.396843194961548, -0.2933540940284729, -0.18719452619552612, 0.0524897538125515, 0.9205911159515381, 1.7140519618988037, -0.3593604266643524, 2.219280242919922, 0.5377029776573181, 0.06068594008684158, -0.9461219310760498, -0.16645191609859467, 1.5460656881332397, -1.7814687490463257, 0.1017579659819603, 0.631578803062439, -0.4345226585865021, 0.49427467584609985, -1.0574253797531128, -1.5021356344223022, 0.06608245521783829, -2.166550636291504, 0.6852281093597412, -1.1681674718856812, 0.20286938548088074, 0.32120686769485474, -0.6787460446357727, 0.2904095947742462, 0.9030964374542236, -0.0633198544383049, -0.7806010246276855, -0.29861536622047424, 0.65696781873703, -0.06953359395265579, 0.48154839873313904, 0.31266671419143677, -0.20981904864311218, 1.4463322162628174, 0.8566976189613342, 0.21929465234279633, 0.4808100759983063, 0.41595175862312317, -0.014233370311558247, -0.2700195610523224, 1.8429145812988281, 1.3579639196395874, 0.7533789277076721, -0.989749550819397, 0.7713146805763245, -1.414799451828003, -0.9398747682571411, 0.9406393766403198, -0.8852676153182983, 0.11399899423122406, -0.09712854772806168, -0.832804262638092, 0.6802981495857239, -0.34988728165626526, -1.969762921333313, -0.43291977047920227, 0.9061086177825928, 0.8838029503822327, -1.7460174560546875, 0.25922515988349915, 0.21992817521095276, -0.48194757103919983, -1.341658353805542, -0.2679000794887543, -0.0049408674240112305, -0.46736958622932434, 1.3587241172790527, -0.9510694742202759, -0.47052037715911865, 0.6762376427650452, 0.3326651453971863, 0.5379707217216492, -0.3600950837135315, -1.380323052406311, 1.1979923248291016, 0.3965604901313782, 0.2691832184791565, 0.9971886873245239, -0.18517716228961945, 2.219683885574341, 0.7436943650245667, -0.7378569841384888, 1.8532209396362305, -0.03980045020580292, 0.8079771399497986, -0.5243882536888123, 0.6331892013549805, 0.3362240791320801, -1.0797981023788452, -0.7458600401878357, 0.3155857026576996, 1.1419525146484375, 1.868233323097229, 1.7345176935195923, -1.0096919536590576, 0.431824654340744, 0.4929307699203491, 0.5979518890380859, -0.3116719126701355, -0.40348923206329346], [-0.6493980884552002, -1.2313076257705688, -0.15330950915813446, 0.151123046875, 0.028427893295884132, 0.10936063528060913, -0.16750335693359375, 0.7709605097770691, -3.7131946086883545, 1.7922718524932861, 1.0178897380828857, 0.008421370759606361, 1.6079857349395752, 0.13817648589611053, -0.6833368539810181, 0.087992824614048, 0.20451244711875916, -0.005334703717380762, 0.40824928879737854, -1.2435680627822876, -0.004096247721463442, 0.7777735590934753, 0.581239640712738, 0.33214402198791504, 1.039829969406128, 0.7874294519424438, 0.41334906220436096, 0.8416383862495422, 2.4396839141845703, 1.1085566282272339, -0.005371507722884417, -1.122847080230713, 0.2506505250930786, 1.117538571357727, 1.0013700723648071, -0.8800936341285706, -0.9210155606269836, -1.1063144207000732, -0.17341478168964386, -1.1279152631759644, 0.9006671905517578, 1.02829110622406, -1.8481101989746094, -0.9202346801757812, 1.582478404045105, 2.2932822704315186, 0.1572231650352478, 0.4116058349609375, 0.9056976437568665, 0.362693190574646, 0.9676489233970642, 0.7465802431106567, 0.5608088970184326, 1.4733223915100098, -0.31890904903411865, 0.38800013065338135, 1.2889324426651, -1.1102663278579712, 0.07087210565805435, -0.5198588967323303, -0.17685893177986145, -0.365958571434021, -0.007983208633959293, 0.06304612755775452, -1.2930783033370972, -1.8248964548110962, -0.9325624704360962, -0.5122173428535461, -0.4365275800228119, 0.12884213030338287, 0.4009254276752472, -1.1211344003677368, -0.37784233689308167, -0.0937456265091896, 0.7564108967781067, -0.8630189299583435, -1.1054210662841797, 0.0711621642112732, 0.11285143345594406, -1.2829797267913818, 1.5123558044433594, -0.09951838850975037, 1.3491706848144531, 0.38974806666374207, -0.17619763314723969, -1.0042123794555664, 0.023090828210115433, 1.1095598936080933, 0.5248405933380127, 0.5506985187530518, 1.1635178327560425, 0.4359854459762573, 0.869935929775238, 0.7290284037590027, -0.5458539128303528, -0.07736720144748688, -0.8864478468894958, 0.6564632058143616, -0.7342329025268555, -0.04358510300517082, -0.7333071827888489, -0.7949109077453613, -0.7310415506362915, 0.3270404040813446, -1.5783988237380981, -0.1245674416422844, 0.7444038987159729, -0.1220511794090271, 1.062104344367981, -2.578775405883789, -0.6604065299034119, -1.360554814338684, 1.3179036378860474, 0.4464256763458252, -2.128887891769409, 1.0999279022216797, 0.45799505710601807, 0.4924255907535553, -0.01666150614619255, 0.8360891938209534, 1.4305038452148438, -1.997174620628357, -0.7491733431816101, 0.19089196622371674, -2.8785741329193115, 0.0015800341498106718, -0.0406772643327713, -0.9702373147010803], [0.6293389797210693, -1.2323178052902222, 0.5518689751625061, -0.22086815536022186, 1.6049516201019287, 1.0997450351715088, 0.85740727186203, -0.387428879737854, -0.6133762001991272, 0.6680567860603333, 1.0095361471176147, 2.214853525161743, 2.4572198390960693, -1.4192063808441162, 0.6911953091621399, -0.20914001762866974, -1.8512654304504395, 1.1735568046569824, 0.07114937901496887, -1.5506680011749268, -1.7855993509292603, 0.38749420642852783, -1.9601633548736572, -1.3476855754852295, 1.54068922996521, 0.3368638753890991, -0.5582924485206604, -1.1014782190322876, 0.7437005043029785, 1.1730464696884155, 0.44392573833465576, 0.02380966767668724, -0.7440587878227234, -0.4994112551212311, -0.605566143989563, 0.16248859465122223, 0.7694739103317261, -0.7135207056999207, 1.3883084058761597, 0.2934929132461548, 0.22288395464420319, -0.24134433269500732, 0.23924775421619415, 0.14910773932933807, 0.5698134899139404, -0.09626442939043045, 0.7095438838005066, -0.37508460879325867, 0.30813634395599365, -1.1425281763076782, 1.6797009706497192, 0.5479072332382202, 1.151381254196167, -0.16586270928382874, -0.4230421185493469, 2.32999587059021, 0.6147499084472656, -0.3689141869544983, -0.5116653442382812, -0.14579558372497559, -0.37477588653564453, -1.076573133468628, 1.05698561668396, 1.3260338306427002, -0.08848357945680618, -0.14728526771068573, -0.5989762544631958, 0.8108398914337158, -0.919965386390686, -0.8573310971260071, -0.7034448385238647, -1.0827959775924683, -2.5786807537078857, -0.8355085253715515, 0.4268539845943451, 0.8419243693351746, -1.1450039148330688, -1.165050745010376, 0.7579718828201294, 0.35502251982688904, -2.3052306175231934, 0.09446683526039124, 1.3934288024902344, 1.7678728103637695, -1.3392229080200195, 0.198526531457901, -0.492017924785614, 0.41112831234931946, 0.4730294346809387, -0.13328894972801208, 0.8508749008178711, 1.0288559198379517, -0.8597478866577148, 0.8596762418746948, -0.3003794550895691, 0.02098204754292965, -0.5247394442558289, 0.461501806974411, 0.18564476072788239, -0.36018994450569153, 0.24457788467407227, -0.010805689729750156, 0.4978843629360199, 0.8985664248466492, 0.23204956948757172, -0.5540159344673157, -0.8946111798286438, 0.5525700449943542, 1.2507599592208862, -0.8484118580818176, 0.7266311645507812, 0.6594186425209045, 1.6321860551834106, -0.5728927850723267, -0.1947728842496872, 0.7648059725761414, -2.105863571166992, -0.9914659261703491, 1.1452217102050781, 0.815880537033081, 0.9885439276695251, -0.47867196798324585, -0.9585031867027283, 0.33665210008621216, -1.3517409563064575, 0.7078169584274292, 0.9507821798324585, 1.859220027923584], [0.23256883025169373, 0.14893107116222382, 0.4006834626197815, 0.5695340037345886, -1.1683515310287476, -0.5318135619163513, 1.0440446138381958, 1.1217008829116821, 0.48296311497688293, -1.083972692489624, -0.1219961941242218, 0.7954056859016418, 0.3097434639930725, 0.7094297409057617, 1.7039662599563599, 0.6909621953964233, -0.117024265229702, -1.0056171417236328, -0.05433839187026024, 0.1856183260679245, -0.14738363027572632, -0.11249155551195145, -0.3644011616706848, -1.6413136720657349, 0.07393023371696472, -0.6432235836982727, -0.1340857297182083, 0.4142701327800751, 0.5916518568992615, -0.23550698161125183, -1.819614291191101, 1.2903016805648804, -0.5850750803947449, 0.5028890371322632, -1.0784271955490112, 0.28704842925071716, 0.9181360602378845, 0.2087009996175766, 1.352484941482544, 0.3034522235393524, 0.4614005386829376, -0.40192699432373047, -1.262380599975586, 0.07470215857028961, 0.6408213973045349, -0.9683411121368408, -0.3337589502334595, 0.5936182141304016, -0.17669737339019775, 1.2933703660964966, -0.9539933800697327, 0.5168882608413696, 0.3772916793823242, -1.1843253374099731, -1.1729357242584229, -0.3374764621257782, 0.805872917175293, 0.11751450598239899, 1.6718101501464844, -0.7453253269195557, 0.8311418294906616, -0.5027514696121216, -0.18009856343269348, 0.5648121237754822, 1.9417487382888794, -0.9632051587104797, 2.346919536590576, 0.9173508882522583, -1.5860105752944946, -0.5814631581306458, 0.8185911774635315, 0.6439107656478882, -0.5142782330513, -0.7061656713485718, 0.2936881184577942, -1.8021138906478882, -0.4767981469631195, 0.5637183785438538, -0.8768244385719299, 0.438191682100296, 0.39097362756729126, -1.1031358242034912, -0.16645567119121552, -0.6234481334686279, 1.8551338911056519, -0.6643118262290955, -1.2350611686706543, -1.1180816888809204, -0.23148982226848602, 1.3917555809020996, -0.5157474279403687, -1.072917103767395, 0.8896019458770752, 0.7163299918174744, -1.0060707330703735, -0.6579412817955017, 0.6983143091201782, 0.3178803026676178, -2.2324600219726562, -1.9343633651733398, -0.9235250949859619, -1.4722726345062256, -0.4137290120124817, 0.2142656296491623, -0.7880166172981262, -1.1137346029281616, -0.21381817758083344, 0.514136791229248, -1.661800503730774, 0.854087233543396, -1.9512596130371094, -0.1570800244808197, 0.5129227042198181, -0.8518004417419434, 0.7358405590057373, -1.4720878601074219, 0.058233555406332016, 0.05361436679959297, 0.950714647769928, 0.700525164604187, -0.734490156173706, 0.2664984166622162, 1.1074422597885132, -1.6336740255355835, 0.620231568813324, -1.9318687915802002, -0.03283671662211418, 0.09924031794071198], [-1.205223560333252, 0.8746657371520996, 1.5008940696716309, 2.101803779602051, -0.2896512448787689, 0.5169114470481873, 1.4018222093582153, -0.3937035799026489, -0.2496112436056137, -0.5824032425880432, -0.34858596324920654, -0.5260960459709167, 2.2783477306365967, 0.9681012630462646, -0.8902022242546082, 0.9301971197128296, 1.3722702264785767, -0.2486930936574936, -0.08350320160388947, -2.4223008155822754, 0.037973977625370026, 3.057913064956665, -1.479621410369873, 1.2142537832260132, -0.04487494006752968, 0.6887165904045105, 1.41951322555542, -0.753973662853241, -0.8292925357818604, 0.7087421417236328, -0.20741631090641022, -0.98580402135849, 0.5993011593818665, 0.9748333692550659, -1.21880042552948, 0.5360379219055176, 0.2419537752866745, 0.5656833052635193, -1.164123296737671, 1.1679329872131348, -0.9094507098197937, 0.2710258662700653, 1.1627988815307617, -0.9072065353393555, -0.7428092956542969, -0.8323816657066345, 0.6908984184265137, -0.3799707293510437, -1.8784822225570679, -1.303341031074524, 0.36923977732658386, -1.3181177377700806, -0.6517296433448792, 1.3339786529541016, 0.37735655903816223, -0.19472044706344604, -0.5160084962844849, 0.10707200318574905, 0.520512044429779, 0.7965351939201355, 1.6921442747116089, 0.607329249382019, -1.387486219406128, 0.7472618818283081, 1.779399037361145, 1.1303001642227173, 1.7417670488357544, 0.7718486785888672, 0.5304502248764038, 1.184590458869934, -1.3180676698684692, 1.2084873914718628, -0.5782754421234131, 2.334851026535034, -0.6897886991500854, 0.5165376663208008, -0.7212166786193848, 0.5831676125526428, -2.227019786834717, -1.6831101179122925, 0.6411852836608887, 0.41820892691612244, -0.674322247505188, 0.777671754360199, 1.0515695810317993, 1.165879487991333, 0.3497837781906128, 1.0466750860214233, 1.7201615571975708, 0.5239570736885071, -0.9576976895332336, 0.7498863339424133, 0.4463096559047699, -0.01943812519311905, -0.889672577381134, 0.13989152014255524, 0.15881648659706116, 1.0971689224243164, -0.5455542206764221, 0.6562996506690979, 0.7773661613464355, -1.496648907661438, 0.5211547017097473, 1.131865382194519, 0.36632266640663147, 0.5368903875350952, -0.1482384204864502, -0.14270146191120148, -1.7077014446258545, -0.7957972884178162, -0.056104470044374466, -1.7291065454483032, -0.6712055802345276, 1.2710456848144531, 0.6650951504707336, 1.3448399305343628, 0.19329340755939484, -0.5923711061477661, -0.04186253622174263, 0.07060450315475464, -1.5282355546951294, 0.2361527681350708, 0.25325384736061096, 1.9340097904205322, -0.6967166662216187, 0.23156622052192688, -0.14201056957244873, -2.155380964279175], [0.6336867809295654, -1.0577912330627441, 0.9074109792709351, 0.1246655136346817, -0.546283483505249, -0.10203981399536133, -1.1892659664154053, -1.665223479270935, 0.5280290246009827, 1.8045001029968262, -1.063030481338501, -0.4841597080230713, 0.35007986426353455, 0.6555548310279846, 0.11152864992618561, -0.2629883289337158, -1.3654001951217651, -0.3400450646877289, 0.5938024520874023, 0.08000099658966064, -1.0559091567993164, -0.4623412489891052, -1.124861240386963, 1.3852458000183105, -0.4234565794467926, 0.26778092980384827, 0.3423154354095459, -0.30313965678215027, -1.6381367444992065, -1.224663257598877, -0.07702656090259552, 0.5898023247718811, 2.1750316619873047, -0.592269778251648, 0.3801926374435425, 1.3039379119873047, -0.5029282569885254, -0.21798329055309296, -2.774738311767578, 0.4083634316921234, 0.2102031409740448, -1.6249151229858398, 1.856244683265686, 0.4564313292503357, 0.38780510425567627, -1.4712278842926025, 0.6836977601051331, 0.7574259638786316, -0.46964412927627563, 1.3390802145004272, -0.8165373802185059, -1.6418269872665405, 1.9136805534362793, -1.4225261211395264, 0.38802769780158997, 0.9256726503372192, -1.1087546348571777, -0.18859921395778656, -0.5858135223388672, -1.1190028190612793, -1.3310554027557373, 0.6213472485542297, 0.569987416267395, -0.13699328899383545, 0.202327698469162, -0.3524637520313263, -1.3761920928955078, -0.43787768483161926, -0.14581093192100525, -0.8291597366333008, 0.9368659257888794, 1.046487808227539, 1.3236539363861084, -2.5216596126556396, -0.788587212562561, 0.05915740877389908, -0.5502533912658691, 1.824096441268921, -2.174450635910034, -1.9041059017181396, 0.6246855854988098, 1.253431797027588, 0.6510488986968994, 0.0028234212659299374, -0.8314666748046875, 0.8249068260192871, 1.973860740661621, -0.13490356504917145, -0.01239765901118517, 0.024417953565716743, -1.7525416612625122, 1.5548841953277588, 0.6210021376609802, -2.0300114154815674, -0.41133350133895874, 1.4248450994491577, 1.43108332157135, -0.9268811345100403, -0.1244385689496994, -0.5484350323677063, -1.2814035415649414, -0.7068465948104858, 0.17998063564300537, -0.2659939229488373, -0.8350322246551514, -1.2244744300842285, 1.3132014274597168, 0.580829381942749, 1.6541080474853516, -0.897644579410553, -1.8574998378753662, 0.515155017375946, -0.013003059662878513, 0.3821653723716736, -0.030793646350502968, 0.2852318584918976, 1.6959413290023804, -1.4085383415222168, 0.4368542730808258, -0.49061885476112366, -0.8533781170845032, -0.2776786983013153, 1.093624472618103, -0.0775456354022026, 1.6176011562347412, 2.4766616821289062, 0.6966267228126526, 0.19045038521289825]], [[0.4458027184009552, -1.0283819437026978, 1.135585069656372, -0.10378863662481308, -0.8363596796989441, 0.34711411595344543, 0.5954790115356445, 1.007624864578247, -0.35000303387641907, -0.19299082458019257, 1.9126474857330322, -1.2812443971633911, -2.0306379795074463, -0.03802497684955597, 0.049854688346385956, 1.348982334136963, 2.6276683807373047, 0.6380600333213806, 0.08526083081960678, -0.6532680988311768, -0.3322790265083313, 0.5639346837997437, -0.9402918219566345, -1.4031275510787964, 0.9574312567710876, 0.40403833985328674, -1.3041274547576904, -1.1852004528045654, 0.14350053668022156, -1.1725398302078247, 0.37369269132614136, 0.04317717254161835, 0.7401252388954163, 1.4231541156768799, 1.6200065612792969, 0.9974777102470398, 0.39548665285110474, -0.7290312647819519, 0.5020437836647034, -1.9650120735168457, -0.14331339299678802, -0.39975178241729736, 0.7888991236686707, -2.027935028076172, -0.7100117206573486, 0.2517658472061157, -1.3376213312149048, -2.3380424976348877, 0.4363122880458832, -0.15387941896915436, -0.7330299019813538, 0.5625481009483337, -1.3041110038757324, 0.9124846458435059, -1.667775273323059, -0.3374188542366028, -0.23199811577796936, -1.6722126007080078, -0.37631499767303467, -1.7202951908111572, 1.7394657135009766, -1.5639516115188599, -0.3590957522392273, -0.7182158827781677, 0.13934507966041565, -0.12294085323810577, -0.8363928198814392, -0.3231372535228729, 0.9963061809539795, 0.989774227142334, -0.5753482580184937, -0.28158125281333923, 1.8369290828704834, -1.1293270587921143, 1.5309429168701172, -0.43548259139060974, 0.14729110896587372, 0.012357402592897415, -0.4234515130519867, 0.023043598979711533, 0.3511507511138916, 1.0434422492980957, -0.8693686127662659, 0.9757078886032104, 1.7026041746139526, 0.8060442805290222, 0.6893900632858276, -0.11777328699827194, 0.13101354241371155, -0.6161162257194519, 1.1008920669555664, -1.32524573802948, -2.7006962299346924, -0.7517339587211609, -0.6956816911697388, 0.33687636256217957, 0.9340855479240417, -0.998024582862854, -0.19047388434410095, -0.430974543094635, -0.9606636762619019, 1.255135416984558, 0.10969240963459015, -1.0454825162887573, 0.4102937877178192, 1.3340444564819336, -0.3267747163772583, 0.9602270126342773, -0.04541527107357979, -1.0728493928909302, -1.7060884237289429, 1.217468023300171, 2.110417127609253, -0.05146408453583717, 1.0840917825698853, -0.273014098405838, -2.410569906234741, -0.6460942029953003, -0.035966988652944565, 0.24349361658096313, 2.0387046337127686, -1.021526575088501, 0.17816156148910522, -1.2423038482666016, 1.0487674474716187, -0.11315355449914932, -0.2950010895729065, -2.2678487300872803], [-1.081526517868042, 0.04791302606463432, 0.04132792726159096, 1.0531655550003052, 1.5400553941726685, -1.5275547504425049, 1.1251686811447144, -1.098709225654602, 0.5612401962280273, 1.2927011251449585, 0.5246246457099915, -0.9797285795211792, -1.138528823852539, 1.4125488996505737, -0.0675521194934845, -3.084489345550537, 0.9863253831863403, -0.84250807762146, -1.2822531461715698, -1.8793431520462036, 0.2596978545188904, 0.8780404329299927, 0.13647598028182983, -0.13711757957935333, -1.3238276243209839, -0.6392645239830017, -0.22721943259239197, -1.0486661195755005, -0.9310252070426941, 0.500286340713501, -1.220815658569336, 1.163573145866394, -0.370231956243515, -0.9036391973495483, -0.6362432241439819, 0.9949620366096497, -0.371695339679718, 0.0092695914208889, 0.6897396445274353, 1.022669792175293, 1.1838109493255615, -1.1390379667282104, 1.544914722442627, -0.04534444957971573, 0.265318900346756, 0.055691689252853394, -0.6362968683242798, -0.3561178743839264, -1.967560052871704, -0.5005378723144531, -0.28939470648765564, -0.20420575141906738, 0.16615809500217438, -0.29985883831977844, 1.6903141736984253, -1.2492859363555908, 0.2859823405742645, 0.7380738854408264, -0.17770372331142426, -1.5337803363800049, -0.01871579699218273, -0.0058323536068201065, -2.1363203525543213, -1.776632308959961, -0.9757106900215149, 1.7297621965408325, -1.167330026626587, -0.2153216451406479, -1.545021891593933, -1.0365610122680664, 0.11200041323900223, -0.6709129214286804, 0.44363951683044434, 1.9461995363235474, -0.17095516622066498, 0.6045514941215515, -0.14421339333057404, -2.045668840408325, 2.4575424194335938, -0.30841580033302307, 1.3921598196029663, 0.2798124849796295, -0.7740837335586548, -2.2329647541046143, 1.6656123399734497, -0.9719530940055847, 0.5773721933364868, -0.7997457981109619, -1.2756857872009277, -1.4464370012283325, 0.006738088559359312, -0.9101793766021729, -1.6954280138015747, -2.510356903076172, 2.561802864074707, 1.1858879327774048, -1.4945390224456787, -0.24664510786533356, 0.07486917823553085, -1.5578638315200806, 0.949835479259491, -0.4517204463481903, -0.80040442943573, -0.5705481767654419, -0.6249873638153076, 0.09633627533912659, 0.5628093481063843, 1.1218866109848022, -0.29769691824913025, 1.5302050113677979, 0.27544495463371277, -1.3283014297485352, 1.5714002847671509, 0.18159590661525726, 0.31770098209381104, 0.279092013835907, 0.1708674430847168, -0.8438053727149963, -0.7154853940010071, 1.483041763305664, -0.5874866247177124, -0.4113530218601227, -0.9499079585075378, -1.2444145679473877, 0.36596372723579407, -0.10738059878349304, 0.5976548194885254, -1.213290810585022], [-0.97891765832901, 0.012542853131890297, -0.5309109687805176, -1.312806487083435, 1.181004285812378, 1.038059115409851, -1.0314027070999146, 1.6038967370986938, -0.6295678019523621, -0.688546895980835, -0.5123136639595032, -0.5854802131652832, -0.5845038890838623, -0.20586422085762024, 1.586549162864685, -0.6377279162406921, -0.012649618089199066, 0.8488152027130127, 0.31489598751068115, 1.420645833015442, -1.0854357481002808, -0.25093311071395874, 0.8224037885665894, -0.14856526255607605, 1.1943279504776, 0.2016373723745346, -1.9206475019454956, -1.4513932466506958, -0.3826809823513031, -0.6804554462432861, 0.7883105278015137, 0.3950914740562439, 0.5452485084533691, 0.03931621089577675, 0.3600934147834778, 0.0704132542014122, -0.19647745788097382, -0.18177242577075958, -0.3398256003856659, -0.3041257858276367, 1.3649888038635254, 0.7912195324897766, -0.472665399312973, 2.0685880184173584, -0.4126012623310089, -0.39943477511405945, 0.4762687385082245, 0.43519556522369385, -0.8180692791938782, 1.5061715841293335, 0.39779379963874817, -0.2314598709344864, -1.0937318801879883, -0.09490357339382172, -0.1155581921339035, -1.785019874572754, -0.979824423789978, 0.6068147420883179, -0.48032405972480774, -0.02812812849879265, -0.4418031573295593, 1.6213918924331665, -0.9249113202095032, 0.8253446221351624, -0.5817055106163025, 0.2127489298582077, -0.2881511151790619, -0.36073416471481323, -0.29495975375175476, -0.4536573588848114, 0.0936087891459465, -1.0232995748519897, -0.8846229910850525, 0.7791434526443481, -0.7459148168563843, 1.1554288864135742, -0.5326762795448303, -0.4286350905895233, -0.3758213520050049, -1.2441222667694092, 0.549078106880188, -0.7463066577911377, -1.1491074562072754, 0.12304078787565231, -0.35012510418891907, 0.4239519536495209, -0.6289421319961548, 0.18882329761981964, 0.16521242260932922, 0.27115756273269653, -1.0587385892868042, -2.40598464012146, 0.2736189365386963, -0.8392221331596375, -1.3671995401382446, -0.8501628041267395, 0.4142998456954956, -0.06634525954723358, 0.8898608684539795, 0.5249823331832886, -0.6758100390434265, -0.7849746942520142, 0.1057620570063591, 0.03994918242096901, -0.14292116463184357, 0.026161380112171173, -0.13443706929683685, 0.6594139933586121, 0.20500041544437408, -1.528501033782959, -2.5218281745910645, 0.6573927402496338, -0.4621742367744446, -0.2418748289346695, 0.31396275758743286, 0.517876386642456, 0.36134278774261475, 1.6740798950195312, 0.6416143178939819, -0.2230956107378006, 0.6872773766517639, 1.4516664743423462, -0.4044645428657532, -1.3360203504562378, -0.3247254192829132, -0.158417746424675, -0.8232954740524292, 0.6053386330604553], [1.1522083282470703, 0.23766590654850006, -1.007510781288147, 1.6117637157440186, -1.9281424283981323, -0.44452109932899475, 0.7122809886932373, 0.02123354934155941, -0.6020156145095825, -1.5602796077728271, -0.3099902868270874, 1.1952152252197266, -1.1065393686294556, -1.0738416910171509, 0.9644198417663574, 0.934431791305542, 0.5488131642341614, 0.09323418140411377, 0.2524871528148651, 0.4428134858608246, -0.34981241822242737, -0.027468806132674217, -2.770519971847534, 1.0932011604309082, -1.4426994323730469, -1.0798097848892212, 0.039896875619888306, 0.5597468614578247, -0.27510136365890503, 0.214810311794281, 0.976249098777771, 1.4727593660354614, -0.008117209188640118, 0.6823107004165649, 0.8887091875076294, 0.9852649569511414, 2.407097101211548, 0.7901231646537781, -1.1300121545791626, -0.5573325157165527, -1.347179889678955, 0.029068728908896446, -1.5404218435287476, -0.2550828158855438, 1.938854694366455, -1.2411667108535767, -0.6510955691337585, -0.1562599092721939, -0.07743606716394424, -0.2456401288509369, 0.5364480018615723, -0.7135031223297119, -0.43308210372924805, -0.07953181117773056, -0.2181631326675415, -0.6762747764587402, 0.10307929664850235, -0.05553397536277771, -0.4080047011375427, 0.282360315322876, 0.9993587136268616, 0.10788509249687195, -0.3341526985168457, 0.4471781551837921, -0.5702682137489319, -0.3291975259780884, 0.4692729711532593, -0.859216570854187, 1.6668671369552612, 0.5778694152832031, -1.8595521450042725, 1.5957063436508179, 1.1907763481140137, -1.767338514328003, 0.2154482752084732, 0.2144063413143158, -0.697618305683136, -0.22242745757102966, 0.9935781955718994, 0.4075568914413452, -0.43243467807769775, -1.5065381526947021, -0.14054492115974426, 1.0015404224395752, -2.6130282878875732, -0.010108706541359425, 0.3498665690422058, 0.6385656595230103, -1.4465802907943726, -0.06216927617788315, -0.6413744688034058, 0.7105388045310974, 2.465214967727661, -0.4248477816581726, 1.0941121578216553, 1.3878620862960815, 1.5413495302200317, -0.7445233464241028, -0.13125766813755035, 0.5213903784751892, 1.662820816040039, 0.4248719811439514, 0.42675328254699707, -0.8694992065429688, -0.8476598262786865, -1.1213641166687012, 1.13969087600708, 1.4093397855758667, -0.7533064484596252, 0.13036523759365082, 0.3194355368614197, -1.056871771812439, -1.1657726764678955, 1.2814404964447021, 0.2103615403175354, -1.0561400651931763, 0.41728487610816956, -2.3116002082824707, 0.48381465673446655, 0.38531896471977234, 0.20112505555152893, -2.279003381729126, 2.2411587238311768, 0.7154986262321472, 0.15288488566875458, -2.517172336578369, 0.14842373132705688, -0.41290083527565], [-1.4305213689804077, 0.9850857853889465, -1.544784665107727, 0.25221455097198486, 0.6928251385688782, -0.28183767199516296, 0.24134331941604614, -1.6159807443618774, 0.6326747536659241, 0.40870580077171326, 0.8728035688400269, -1.5548584461212158, -1.0759022235870361, 0.254456490278244, -1.0172393321990967, -1.2138382196426392, 0.22162757813930511, 0.10895504802465439, 1.7055046558380127, 1.7820059061050415, -1.105664610862732, -0.12568068504333496, -0.10349558293819427, -0.3566524088382721, -1.0860803127288818, -0.5889172554016113, -1.3340851068496704, -0.9627133011817932, -0.21417731046676636, -0.8274890780448914, 1.374119520187378, 0.14314736425876617, 0.997556209564209, -0.5106388926506042, -0.26499462127685547, 0.2563377916812897, 0.6260653138160706, 0.7880568504333496, -0.4621488153934479, 0.8252560496330261, -0.9928781986236572, -0.6266016364097595, -0.35859835147857666, -0.23770193755626678, -0.8373081684112549, 0.8321434855461121, 0.3651781678199768, 1.1259416341781616, 2.1785054206848145, 0.27305567264556885, 1.3297805786132812, 1.3513885736465454, 0.1740744709968567, -0.7438289523124695, -0.022138996049761772, 0.06416787952184677, 0.7726846933364868, 0.9588916897773743, -0.17757979035377502, -2.222264051437378, -1.2628347873687744, -0.6779358983039856, -0.09648048877716064, -0.6852125525474548, 1.3495054244995117, 0.14131926000118256, 0.5629435777664185, 1.617803692817688, 0.9431907534599304, 0.03041648305952549, -0.5778833031654358, 0.4761662185192108, 0.6376456618309021, -2.4138450622558594, -1.0059996843338013, -0.8835749626159668, -2.923862934112549, -1.7514111995697021, -0.40796342492103577, 0.18855559825897217, -1.5464200973510742, 1.332858681678772, -0.20168545842170715, -1.9531792402267456, -0.043250955641269684, 0.018481530249118805, 0.24522174894809723, 0.19219833612442017, -0.8234087228775024, 1.3746039867401123, 0.018471164628863335, 0.3261876404285431, -0.009924253448843956, -1.024097204208374, -0.8463777899742126, 0.27950963377952576, -1.0163142681121826, 0.15146292746067047, 0.8682944178581238, -0.7000361084938049, -0.6470626592636108, -0.645033061504364, -1.786228895187378, 0.3202599287033081, 1.0517781972885132, -0.34195294976234436, -1.9185889959335327, -0.5417255163192749, -0.37738457322120667, -0.49926134943962097, 0.801876962184906, 0.600981593132019, 2.7864644527435303, 0.4787645637989044, 1.0833088159561157, -0.13861916959285736, -1.2686071395874023, 0.39350661635398865, -0.41200023889541626, -1.3227792978286743, -0.6082165837287903, -1.1422399282455444, 1.2656306028366089, -0.46022963523864746, 1.5655258893966675, -1.8981692790985107, -1.8771010637283325, 1.540131688117981], [1.0157268047332764, 1.6722428798675537, 0.6567541360855103, -0.321544885635376, -0.4934207499027252, -1.9168717861175537, 0.022244486957788467, 0.3726098835468292, -0.7806148529052734, -0.05195047706365585, 0.11240163445472717, 0.4768267273902893, 0.2400268018245697, -2.9405388832092285, -0.10653506964445114, 0.15988749265670776, 0.2574307918548584, 1.3251006603240967, -0.9336191415786743, -0.019938016310334206, 0.7718114256858826, 0.7487861514091492, -0.5887768268585205, 1.5272390842437744, -0.3252948224544525, 0.9448169469833374, 1.0333815813064575, 0.036580368876457214, 0.7960829138755798, -1.1564719676971436, -1.5148464441299438, 0.012946579605340958, -0.7332822680473328, 0.7613650560379028, -0.379823237657547, 0.44759267568588257, -0.22202308475971222, -1.7727409601211548, 0.005182971712201834, 0.7041777968406677, -0.016591321676969528, -0.26878100633621216, 1.1752986907958984, -0.25763365626335144, -0.9973577857017517, -0.3580497205257416, 0.7865586280822754, 0.9244436621665955, 0.7368675470352173, -0.8003333806991577, -0.8787681460380554, -0.9947075247764587, -0.42657670378685, -1.146655559539795, -0.6910462379455566, -0.4817739427089691, 0.3291272819042206, -1.6741865873336792, 1.0675084590911865, 0.04439152404665947, -0.8985655903816223, 1.160575270652771, -1.6142189502716064, -0.14518854022026062, 0.3354414999485016, 0.3754796087741852, -1.4635519981384277, -0.9237402677536011, 0.7889647483825684, 0.6054482460021973, 1.0945266485214233, 2.158247470855713, 0.23842157423496246, -0.01369011215865612, -0.694153368473053, -1.3811291456222534, 1.29851496219635, -0.6873941421508789, 0.6837316751480103, -1.2953894138336182, 0.9174986481666565, -0.5716384053230286, -1.1036908626556396, -1.7021827697753906, 0.9205582141876221, -1.1350346803665161, 0.33299192786216736, 0.25231853127479553, 0.6922970414161682, -0.10993429273366928, 0.41527238488197327, -1.0259631872177124, 0.4121951162815094, -0.23510341346263885, 1.3343064785003662, -0.79063880443573, -0.3479584753513336, 0.5755070447921753, 0.02059665136039257, 0.8182732462882996, -0.3407808542251587, -1.34990394115448, 0.20357485115528107, -1.5067802667617798, 0.8987526893615723, -0.5451750755310059, 0.3171685039997101, 0.47307875752449036, -2.627528190612793, 0.3991156816482544, -1.2415214776992798, -2.336303234100342, 0.20596036314964294, -0.6137285828590393, 0.6545587182044983, -0.33153900504112244, 1.2733007669448853, -0.7721153497695923, 0.3714869022369385, -1.5300897359848022, 0.9766786098480225, 1.0001132488250732, 2.720417022705078, -0.6896906495094299, 0.17011898756027222, -1.6596146821975708, -0.4936509132385254, 0.7731620073318481], [0.8572882413864136, 0.7996627688407898, 1.237614393234253, -0.6883454918861389, 0.3907584249973297, 1.3524948358535767, -1.0795027017593384, 0.8905531764030457, -0.34360024333000183, 0.2629343271255493, 0.06459987163543701, -0.20058603584766388, 0.9629635214805603, -0.8728265166282654, 0.7348907589912415, 1.4191687107086182, -0.34103676676750183, -0.8849092125892639, -0.5092291235923767, -0.011065891943871975, 0.06619185954332352, 0.8319929242134094, -0.30551379919052124, 1.5912691354751587, -0.7005488276481628, 1.3255456686019897, -0.7190083861351013, 0.9649685621261597, 0.28811535239219666, 0.19601592421531677, -0.9972835779190063, -1.1634193658828735, -0.0036925103049725294, 0.5447530150413513, 0.3121466636657715, -0.10754667222499847, 1.379296064376831, 0.48535487055778503, 0.9736064672470093, 0.18523797392845154, -0.12400510907173157, 0.04643448442220688, 0.4674699306488037, 0.40162143111228943, 1.2397220134735107, -1.022302508354187, 0.21277053654193878, -0.20547419786453247, -0.485539048910141, 0.2624278664588928, 0.7278181910514832, 1.2472583055496216, -0.012671149335801601, -0.18344931304454803, 0.997628390789032, 1.2898372411727905, -1.7161961793899536, 1.521254539489746, -0.3612043559551239, 2.0480875968933105, 0.34844839572906494, 0.9063772559165955, -0.26470857858657837, -0.28134989738464355, 2.8547897338867188, -0.3250826895236969, -0.7043558359146118, 0.9662269949913025, -0.03457478806376457, -0.5369156002998352, -1.1487749814987183, -0.9209675788879395, -0.34329453110694885, -0.020615125074982643, 0.22158659994602203, -0.3602673411369324, -0.8447570204734802, 1.042002558708191, 0.24346986413002014, -0.9013445973396301, 0.7308520078659058, 1.3635023832321167, -1.8958176374435425, -0.9432766437530518, -0.2129642814397812, -0.7299814820289612, -0.8898531198501587, -0.12257307767868042, 0.8600631952285767, 1.3468642234802246, 0.5970821380615234, -0.4307743310928345, 0.9293835163116455, -0.6240617036819458, -2.5475401878356934, 0.3291270136833191, -1.1066455841064453, -0.3512503206729889, 1.1628199815750122, 0.01791311241686344, -0.15805354714393616, -0.2929477393627167, 1.7305278778076172, -0.6746214628219604, 0.21171920001506805, 0.5954041481018066, 0.8300994634628296, 0.6106204986572266, -0.8765093684196472, 2.7238242626190186, -0.48485592007637024, 1.744857907295227, 0.6190007925033569, 1.0488907098770142, -1.4451370239257812, 1.1908537149429321, -1.0414416790008545, -1.203573226928711, 0.8059486150741577, 0.09080152958631516, 0.2632005512714386, 1.5781605243682861, 0.8072377443313599, -1.7876356840133667, -1.0253815650939941, -0.39717456698417664, -0.2102382779121399, 1.1179699897766113], [0.32243064045906067, -0.378192663192749, -0.5285894274711609, -2.128201723098755, 0.36218076944351196, -0.1197495311498642, 0.4167740046977997, -0.48233863711357117, 0.6399008631706238, 1.7297767400741577, -0.4308115541934967, 0.49929824471473694, 0.7323814034461975, 0.596572995185852, 0.38938164710998535, -0.7095112800598145, -2.0533103942871094, 2.55928111076355, -1.600387454032898, 0.03995034471154213, -1.3589903116226196, -1.7996573448181152, 1.0530210733413696, -0.06107249855995178, 0.6248251795768738, -0.031178191304206848, -0.5748886466026306, 0.7802256345748901, -0.1338592767715454, -0.37785711884498596, 0.9179219007492065, 0.5981959104537964, -1.0081267356872559, -0.3245401084423065, 1.4998825788497925, -0.25415945053100586, 0.562403678894043, 0.3057151734828949, -1.5419435501098633, -0.7092097401618958, -0.8267688751220703, 0.6362596154212952, -0.5137919187545776, 0.841139018535614, -1.4543633460998535, 0.621470034122467, -0.5309397578239441, 0.004225737880915403, -0.0996021032333374, 0.24589803814888, -0.40481600165367126, -0.0942574143409729, 0.5062688589096069, 0.9795712828636169, -0.21020659804344177, -1.0328763723373413, -0.3572502136230469, 0.9195497035980225, 0.33659082651138306, -1.400484323501587, 1.514973759651184, -0.5288515686988831, -0.8521987199783325, -0.10685118287801743, -1.2089110612869263, -0.805338978767395, -0.8262713551521301, -0.7164143919944763, 0.5654771327972412, -1.6238433122634888, -0.26373475790023804, -0.5520431399345398, -0.30549904704093933, 1.804882287979126, 0.12767574191093445, -0.9110307097434998, -0.159064382314682, 0.5856339335441589, 0.31236031651496887, 0.7808414101600647, 1.4542726278305054, -1.1497999429702759, 1.0675803422927856, 0.3231244385242462, 0.25791943073272705, -0.8689178824424744, -2.3134348392486572, 1.492105484008789, 0.030557911843061447, 2.1233129501342773, -1.1163232326507568, 0.006842960137873888, -2.7841897010803223, -0.21164391934871674, 0.6508018374443054, 0.05823246389627457, 0.8148357272148132, 0.39887478947639465, 0.9036808609962463, 0.16721321642398834, 1.810595154762268, 1.3260000944137573, -1.0241373777389526, 1.2774959802627563, -0.9749280214309692, 0.8037851452827454, 0.7373956441879272, -1.0700948238372803, 0.17082151770591736, 1.1818609237670898, 0.4293144941329956, 1.7261645793914795, -0.21056398749351501, 0.9324939846992493, 0.29033198952674866, -0.10694015771150589, -0.1057049110531807, -0.7792219519615173, -1.6095755100250244, 0.8555065989494324, 0.08767063170671463, 0.5385749340057373, 1.3211240768432617, -0.8777473568916321, 0.26867035031318665, 0.8849883079528809, 0.10455562174320221, 0.24707430601119995], [0.4952540993690491, -0.8229900598526001, 0.8481098413467407, -0.16068805754184723, 0.7750614285469055, 1.2288644313812256, -1.889620304107666, -0.20500805974006653, -1.4741332530975342, 1.9668444395065308, -1.6794906854629517, -0.6236470341682434, -1.3985902070999146, 0.2251153439283371, -0.05894654244184494, -0.4403003454208374, -2.2209415435791016, 0.6826305389404297, 0.6803534030914307, 0.5204090476036072, -0.32203209400177, 0.24822671711444855, -0.721235454082489, 1.2537204027175903, 0.12407614290714264, -1.3656890392303467, 1.4161436557769775, 0.3019334375858307, -2.538654327392578, -0.6610612869262695, -0.4506371319293976, -0.8008161187171936, -0.8114335536956787, -0.15961064398288727, -0.48095208406448364, 0.3674888610839844, 0.7149649858474731, 1.0601806640625, -0.31162574887275696, -0.7580145001411438, -0.2565491795539856, -0.7828466892242432, -0.20890751481056213, 1.3561220169067383, -0.30570119619369507, 1.1410009860992432, -1.1842241287231445, 0.3276779353618622, -0.03630969673395157, -0.6618837118148804, -0.03751785680651665, 0.21551421284675598, -0.5953649878501892, 1.3617959022521973, 0.07530467957258224, -1.2831733226776123, -1.6620248556137085, -0.6844454407691956, -0.9547751545906067, 0.9189595580101013, -0.6710183620452881, 0.897192120552063, -0.8394454121589661, -0.17939478158950806, 1.1586925983428955, 0.29195356369018555, 0.47025468945503235, -0.7324402332305908, 0.37747272849082947, -0.9038589596748352, 0.8748572468757629, 1.1176098585128784, -0.019422685727477074, -1.6037943363189697, 0.1630096733570099, 0.8977072238922119, -0.3998173177242279, -0.07910261303186417, -1.4367084503173828, 1.704036831855774, 0.5430723428726196, 0.5670189261436462, -0.11858785897493362, 1.1144546270370483, -2.12099552154541, 0.4081701636314392, -0.22132307291030884, 1.6372196674346924, -1.3497676849365234, -1.4218370914459229, -0.25054776668548584, -0.3295769691467285, 0.896569550037384, -2.0516366958618164, 0.3989403545856476, -0.6481990814208984, -0.06231505051255226, -0.12276842445135117, 2.179494857788086, 0.1523256003856659, -1.36507248878479, -1.8743383884429932, -0.4442390501499176, 2.5691683292388916, 0.3263448178768158, -0.6067237257957458, 1.0582653284072876, -1.01140296459198, -1.209800362586975, 0.09851885586977005, 0.14479538798332214, -1.6116628646850586, -1.7056993246078491, 1.4308652877807617, -0.337544709444046, 0.5247745513916016, 1.0854331254959106, -1.0614001750946045, -1.2781472206115723, -1.0534899234771729, 0.4574175179004669, 1.154928207397461, -0.5964782238006592, 0.5714492201805115, -0.9788039922714233, -0.12813033163547516, -0.23400253057479858, 0.0199015811085701], [0.06566189229488373, 0.9823886156082153, -0.5837622284889221, 0.4825133681297302, 0.006717159878462553, -0.40916457772254944, 0.11233635991811752, -0.7479186058044434, 1.418508529663086, 1.4738409519195557, -0.6257026195526123, 0.12794236838817596, 1.8718235492706299, 0.7106302976608276, 1.0026788711547852, 1.3584235906600952, 0.6664099097251892, -1.2050344944000244, 0.7465949654579163, 1.083858847618103, 0.7234677076339722, 1.8421354293823242, -1.9566067457199097, 0.06887708604335785, -1.2000833749771118, -1.158347487449646, 0.6771380305290222, -1.4935590028762817, 0.002877660095691681, 0.7634961605072021, -0.39116933941841125, 1.2208796739578247, 0.8598788976669312, 0.15595941245555878, 0.157679945230484, -1.0754903554916382, -0.8086519241333008, -1.6880985498428345, 0.24268394708633423, 0.21312810480594635, -1.4011867046356201, -1.1883848905563354, 1.4679633378982544, 2.310218334197998, -0.7978174090385437, 0.2145487517118454, 0.8071337342262268, -0.03048648312687874, -1.0634039640426636, 0.09887980669736862, -1.2324628829956055, 1.0264863967895508, 0.6642083525657654, 0.19465504586696625, 0.3301868736743927, 0.8617329597473145, -0.5141904354095459, 0.025681452825665474, 0.0925200954079628, -1.2088756561279297, -0.9396334886550903, -1.0723236799240112, 0.5957211852073669, -0.45647042989730835, -0.27604663372039795, -0.3398253619670868, 1.642820119857788, -2.2269210815429688, 0.3372373878955841, -2.4744880199432373, -1.0693459510803223, -0.5591049194335938, 0.8445087671279907, 1.4465981721878052, 0.44392848014831543, 1.4745733737945557, 0.7922787070274353, 0.4643200933933258, -0.6754891276359558, -1.0290420055389404, -1.837836742401123, -0.11168330907821655, 1.3418322801589966, 0.3204095661640167, 1.779421329498291, 0.8047055006027222, -1.5421103239059448, -1.1429792642593384, 0.17404146492481232, 0.8317531943321228, -0.6397640705108643, 1.1145169734954834, -0.5951390266418457, 2.042937994003296, 0.2112732231616974, 0.48846399784088135, -1.1553033590316772, 0.5888042449951172, -0.24562007188796997, 0.30831772089004517, -0.5344407558441162, 1.3285318613052368, -0.8845054507255554, 0.9886547327041626, 1.5222587585449219, 0.4345313012599945, 1.2428325414657593, -0.2041700929403305, -0.36272361874580383, -0.26609528064727783, 1.7054916620254517, -0.3497723340988159, 0.6730057597160339, 0.04813512787222862, 2.596071720123291, 0.14010223746299744, -0.47922202944755554, -1.345202088356018, -0.9889475703239441, 0.9243924617767334, -1.3562110662460327, 0.2978177070617676, -0.9510555863380432, 0.17643383145332336, 0.9323015809059143, 0.08487568795681, -1.490570306777954, 1.2323429584503174], [1.5985051393508911, 0.6675412654876709, 0.8954570293426514, 0.44828900694847107, 0.2356363981962204, -0.9118844270706177, -0.6869239211082458, 0.0696030855178833, -0.4274095892906189, 0.5731375217437744, 0.36718565225601196, -1.831295132637024, 0.18977123498916626, 0.6940325498580933, 0.13635464012622833, -0.7053871750831604, 0.03037753328680992, 0.2584505081176758, -0.8773932456970215, -0.9361591935157776, -0.6991045475006104, -0.0710032507777214, -3.2131404876708984, 1.0451358556747437, 1.8471988439559937, -0.41842472553253174, 0.9437156319618225, 0.24137429893016815, 1.2124371528625488, 2.682915687561035, -2.395504951477051, 0.19943886995315552, 0.02393517643213272, -0.8867297172546387, 0.9380335807800293, 0.061114389449357986, 0.2330699861049652, -0.3632524609565735, -0.11485227197408676, -0.11679185926914215, 0.31325703859329224, 1.2935302257537842, -1.9525409936904907, 0.12389253824949265, 0.46956613659858704, 0.21797296404838562, -0.7041618227958679, -0.5986332297325134, 0.8379034399986267, 0.6675257086753845, -0.7910752296447754, -0.02733132429420948, -0.3089694380760193, -2.3422276973724365, 1.7225021123886108, -0.8709638714790344, -0.5480535626411438, -1.958229660987854, 1.728871464729309, 0.4906885623931885, -0.36298245191574097, 0.794192910194397, -1.082543134689331, 2.383164167404175, 1.5001835823059082, 0.20804476737976074, -1.9406999349594116, 0.9277804493904114, -0.7109812498092651, 0.7970671653747559, 1.4462456703186035, -0.39266106486320496, -0.9943546652793884, -0.3364750146865845, 1.8027859926223755, 0.7442723512649536, -0.48625555634498596, 0.17945550382137299, -0.48477432131767273, -1.4451854228973389, -0.06024865061044693, -0.5088666081428528, -0.4854172468185425, -0.4352567791938782, 1.6156737804412842, -0.05050528794527054, 0.871599018573761, 1.3552422523498535, -0.7649535536766052, 0.17077597975730896, -0.41962289810180664, -1.5511888265609741, -0.19344158470630646, -1.2389545440673828, 1.0763590335845947, -0.45297175645828247, -1.514731764793396, 1.5268758535385132, -0.183091938495636, 1.0409985780715942, -0.30079731345176697, 0.600103497505188, -0.9128246307373047, -1.7189412117004395, 0.7741384506225586, 2.056995153427124, 0.15546022355556488, -0.48382335901260376, -2.435889959335327, -0.12744101881980896, 0.6680161952972412, 0.15058259665966034, -1.6314486265182495, 0.5022651553153992, -0.08587358891963959, 0.2168189436197281, 0.65631103515625, -0.5780384540557861, -1.4420976638793945, -1.3524467945098877, -1.4555562734603882, -0.431613564491272, -1.212770700454712, -0.5071077942848206, 0.9496694803237915, 1.2228448390960693, 0.7304155230522156, -1.8598911762237549], [0.4324844479560852, -0.792647123336792, -1.4139630794525146, -2.2459306716918945, 0.5846719741821289, -0.24267785251140594, 0.7115607857704163, 0.8766079545021057, -0.6691173315048218, -0.09644369781017303, 0.020542120561003685, -0.621563732624054, 2.000890016555786, -0.0796966627240181, -0.2682553231716156, 0.48936232924461365, -0.7815732359886169, 0.8580960035324097, -0.375966876745224, -1.5350706577301025, -1.1261874437332153, -1.9222947359085083, 2.0492215156555176, 0.6956661343574524, -0.5488405227661133, 0.09195692092180252, -0.5174410343170166, 1.3406691551208496, 1.475496530532837, -0.913922905921936, 0.7133135795593262, -0.48919424414634705, -0.7937911748886108, -1.8231197595596313, 1.6518124341964722, -2.398854970932007, 1.0367916822433472, 1.0603753328323364, -0.6300629377365112, -1.1758490800857544, -0.39416244626045227, -0.938450813293457, -0.21488195657730103, 0.749602735042572, 0.7023699283599854, -0.3973303437232971, -1.798934817314148, -1.3761123418807983, -2.257145643234253, -2.333230972290039, -0.019588707014918327, -1.0024501085281372, -0.054782744497060776, 0.17766070365905762, -1.8251301050186157, -0.08406069874763489, -0.6005679965019226, 0.44177547097206116, -0.9291331768035889, -2.1149561405181885, -1.9887685775756836, 0.16042765974998474, 0.555675745010376, 0.6847162842750549, 0.810703694820404, -1.3347439765930176, -0.6477590799331665, 1.7039477825164795, 1.8763843774795532, -1.8971794843673706, -0.10576247423887253, 1.5022153854370117, 0.9426578879356384, -1.1576392650604248, -1.0458835363388062, 0.8774108290672302, 0.706243634223938, 0.08760018646717072, 1.107447624206543, 0.0013854712015017867, 0.2703346312046051, -0.34680041670799255, 1.7122044563293457, 1.9930763244628906, -1.4701300859451294, 0.9361233711242676, -0.25684571266174316, -1.8241093158721924, -1.0592211484909058, 0.11782687902450562, 1.027502179145813, 0.2218647599220276, -0.5260317325592041, 1.4959094524383545, -0.34522897005081177, -0.44935452938079834, -0.22344039380550385, 0.4621965289115906, 0.6483927965164185, 0.2805347442626953, -0.8696252703666687, -0.2946432828903198, 0.7480164766311646, -0.7883750200271606, -1.1884747743606567, 0.3501370847225189, 0.6271457672119141, 0.9100930094718933, 1.2414441108703613, 0.08978238701820374, -1.1663906574249268, -0.8296836018562317, -1.2461857795715332, -0.9146921634674072, 0.23944009840488434, 1.0697101354599, 1.0218329429626465, -1.032107949256897, -0.21878068149089813, -1.1879483461380005, 0.4259697198867798, -2.1066768169403076, 0.5002670288085938, 1.8318206071853638, 1.2733218669891357, 1.2607629299163818, -0.7791721224784851, 0.4527552127838135], [1.1464968919754028, 0.6670555472373962, 1.1212759017944336, -0.39100342988967896, -0.5686046481132507, -0.8591768741607666, 1.4539146423339844, -0.19271086156368256, 0.014738354831933975, 0.0010125605622306466, -0.39898964762687683, -0.1928071826696396, 0.9147784113883972, 0.2608092427253723, -1.3600839376449585, -2.4964404106140137, -1.0153334140777588, 0.0950809195637703, -0.6275466680526733, -0.0692739337682724, 1.41769278049469, -0.5591451525688171, 1.0295629501342773, -0.1469242423772812, -0.008102660067379475, 1.2646832466125488, -0.055055104196071625, 0.17903529107570648, -0.45012083649635315, 2.3430871963500977, 0.751488447189331, -0.09972747415304184, 0.48401349782943726, -0.750560462474823, -1.8273489475250244, 1.246109962463379, 0.41517311334609985, 1.1029410362243652, 0.16089802980422974, -0.014651741832494736, 1.7312291860580444, 0.7518895268440247, 1.1504058837890625, -0.05235759913921356, 0.17808188498020172, -0.8730559945106506, 0.8746969699859619, 0.6245944499969482, -0.06319644302129745, 1.5998187065124512, -0.003264226485043764, -1.4525998830795288, 0.6981346607208252, -0.9051201343536377, 1.0683233737945557, -0.4892086386680603, -0.13767027854919434, -0.6320529580116272, 0.5891929268836975, 0.30759432911872864, 0.7633026838302612, 0.2465706616640091, -1.5900200605392456, -0.7057018876075745, 0.35132426023483276, -1.3304579257965088, 0.4749244153499603, -0.17832085490226746, 0.6031761169433594, 0.3070162236690521, 1.407021164894104, -0.970178484916687, 1.2374637126922607, 0.9712494015693665, -0.5846466422080994, 0.855696976184845, -1.293290138244629, -0.34172454476356506, -1.7574836015701294, -0.7146323323249817, 0.5720076560974121, -0.3483928442001343, -0.1273108422756195, 0.1255018413066864, 0.45751842856407166, 0.46559831500053406, 1.639543056488037, 2.4846253395080566, 0.12195437401533127, 0.7040272951126099, -1.164451241493225, 0.14566127955913544, 1.372004747390747, -0.4880027174949646, -0.36596354842185974, 1.022994875907898, -0.25552403926849365, 0.022077735513448715, 1.0451722145080566, 0.1642598658800125, 0.2697099447250366, 0.11735977232456207, -0.7114752531051636, 0.12120706588029861, -0.17799831926822662, 0.2779588997364044, 0.6069995164871216, -0.5566418766975403, 2.1027698516845703, -0.48756325244903564, 1.087838888168335, -0.9550240635871887, -1.2557905912399292, -0.9269726276397705, -1.4558765888214111, -0.6364315748214722, 0.9150776863098145, -0.21250572800636292, 1.6400498151779175, -0.7859843373298645, -0.9624824523925781, -0.6055914163589478, -0.38664597272872925, -1.062795877456665, 1.273759126663208, -0.21826237440109253, -0.6943802833557129, -0.6833365559577942], [0.1191035583615303, 0.12202112376689911, -0.05792318657040596, 1.5542329549789429, -1.3848732709884644, -0.1776265799999237, 0.8255395293235779, -0.646270751953125, 0.28824886679649353, 0.2791723310947418, 0.7785665988922119, 0.7195281386375427, -0.5227752327919006, -0.35750457644462585, 0.5675451159477234, 2.8218207359313965, -0.4598884880542755, -1.020370602607727, -1.1423166990280151, -0.693792998790741, 0.18220794200897217, -0.21405541896820068, -1.0835859775543213, -0.8299993872642517, 2.094344139099121, -1.0761897563934326, -0.8797637820243835, -1.616703748703003, -0.23979398608207703, 0.42846933007240295, -0.07937854528427124, -1.4016379117965698, 0.6235489249229431, -0.9460666179656982, -0.09627612680196762, -0.7123708128929138, 0.6821483373641968, 1.0143561363220215, -0.5032765865325928, 1.1760928630828857, -1.2900587320327759, -0.2109593152999878, -0.7988370060920715, -0.8012750744819641, -1.6608777046203613, -0.23047958314418793, 0.7147901654243469, -0.26000097393989563, -0.17955447733402252, -0.16139207780361176, -0.46101635694503784, 1.471369981765747, 0.06443800032138824, -0.5825226306915283, 0.7168212532997131, -0.5853408575057983, -0.5135992765426636, -0.1558127999305725, 1.199226975440979, -1.0699530839920044, 0.639892041683197, -0.852985680103302, -0.9910719394683838, 1.1058897972106934, 0.6000170707702637, -0.8647775650024414, 1.5988543033599854, -1.0670015811920166, -0.06993795931339264, 0.8080668449401855, 0.8778502345085144, 0.0434102863073349, 0.9484547972679138, 0.2138155847787857, -1.3441756963729858, -1.4821457862854004, 0.8936394453048706, -0.1170634999871254, 0.7759057283401489, 0.13283702731132507, -0.20756901800632477, 0.7642858624458313, 0.2301444709300995, 1.2950174808502197, 0.022950217127799988, -2.313843011856079, -0.1941770613193512, 0.4431821405887604, 1.187038779258728, -1.3477838039398193, -0.10104217380285263, -1.0216041803359985, -1.2753475904464722, 1.1951204538345337, -0.9439117312431335, -2.4246342182159424, -0.5658367276191711, -0.8055711388587952, 0.14758923649787903, -0.8195569515228271, -1.6171215772628784, 1.306610345840454, -1.1656988859176636, 0.6151076555252075, -1.3037573099136353, 1.3833194971084595, -1.5106796026229858, -1.2454479932785034, 0.07028304040431976, 0.6586415767669678, -0.14256875216960907, -0.8890514969825745, 0.4801122844219208, 0.01406499557197094, -0.4265094995498657, 1.2857991456985474, 2.8412253856658936, 1.079277753829956, 0.1422324776649475, -0.05374389886856079, -0.18144698441028595, 0.2126135379076004, 1.9598350524902344, 0.42862841486930847, -0.5217560529708862, 0.7627915740013123, 0.06690862774848938, -0.4350057542324066], [1.0856409072875977, 0.0979846939444542, -0.40129807591438293, -0.8161816000938416, 0.15577849745750427, 0.7479727268218994, 2.437509059906006, 0.5990975499153137, -0.2536923587322235, 0.33802467584609985, -2.222991466522217, -0.3151077330112457, -1.0136317014694214, 1.3244948387145996, 0.10115023702383041, 1.2071703672409058, 0.47735655307769775, 1.963185429573059, 0.19915851950645447, 0.48708871006965637, -1.2473148107528687, -1.3691866397857666, 0.10169260203838348, 0.1842227727174759, -0.4723515510559082, -0.7692748308181763, -0.16852784156799316, -0.3436146676540375, 0.47471460700035095, -0.6508891582489014, -0.5030545592308044, 1.616377830505371, -0.8685182332992554, 1.0052013397216797, -0.4842222034931183, 0.2156795859336853, -0.7868064045906067, 1.8115918636322021, -1.6780662536621094, 0.10802211612462997, 0.5938643217086792, -2.1942367553710938, 0.027790384367108345, 0.4728771150112152, -0.11978999525308609, 0.8432317972183228, -0.8324607014656067, -0.9737804532051086, -1.551377534866333, -2.0386276245117188, 1.0868943929672241, -1.4444708824157715, -2.6401994228363037, -0.859987735748291, 0.22902750968933105, 0.30998530983924866, 0.7798300385475159, 0.6944396495819092, -0.03249664232134819, 0.5046732425689697, -0.4487013816833496, -0.5893266797065735, 0.4565228223800659, -2.0101184844970703, -0.626681923866272, 0.8745743632316589, -0.7512755393981934, -1.3383047580718994, -1.2399840354919434, -1.97858464717865, -2.229562520980835, -0.5823709964752197, 1.1039652824401855, 0.8864845633506775, 2.0423781871795654, -0.1802673041820526, -0.4526301622390747, 0.5912826657295227, 0.09354996681213379, 0.03276621922850609, -0.3079793453216553, -1.021004319190979, -0.46714040637016296, -0.5056672692298889, -1.6436452865600586, 2.4821863174438477, 0.7501358389854431, 0.37258023023605347, 0.29603782296180725, 1.4836502075195312, 2.211568832397461, -0.2790719270706177, 1.3293150663375854, 0.5965914726257324, 1.1713776588439941, 0.1919826716184616, -1.9667900800704956, -0.6801121234893799, 0.09160242229700089, -0.2362155169248581, 3.126328229904175, 0.7846126556396484, -0.3267998993396759, 0.05593560263514519, 1.4859542846679688, -0.9590994715690613, -1.321641206741333, -0.4547581970691681, 1.1102274656295776, -0.22429490089416504, 0.8469089865684509, 1.5101442337036133, 1.7746901512145996, -0.22129693627357483, 1.8193449974060059, 0.3505043089389801, -1.2027510404586792, 1.4323015213012695, 0.6752868890762329, -0.3536672294139862, -1.019453525543213, 0.036054957658052444, 0.6112858057022095, -0.22679071128368378, -0.22988811135292053, -0.7360263466835022, -0.23514018952846527, 0.334099680185318], [1.6723819971084595, -0.37713247537612915, 1.0276466608047485, 0.6802226305007935, -0.5131598114967346, 1.4600236415863037, 1.2191435098648071, 0.2397104799747467, 1.998529314994812, -0.7193592190742493, -0.33501437306404114, 1.8480581045150757, -1.6455130577087402, -0.02518981695175171, 1.0220104455947876, 1.0186822414398193, 0.1976354569196701, 0.9193295240402222, -0.4419349133968353, -0.35157930850982666, 0.3978557288646698, 0.7822408080101013, -0.7378278374671936, -1.124495506286621, 0.2316892296075821, 0.07662545889616013, 0.6998704671859741, -0.2952214181423187, -1.3542020320892334, -0.8079032301902771, -1.6589659452438354, -0.035262331366539, 0.6819707751274109, -0.09525739401578903, -0.29719600081443787, -0.85938960313797, 1.300847053527832, 0.4176872670650482, -0.1426277458667755, -1.5612516403198242, -0.34506872296333313, 0.31926172971725464, 0.300008088350296, -1.2182655334472656, 0.01295387838035822, 0.4740768373012543, -1.2409512996673584, -1.2303955554962158, -1.0212626457214355, 1.134512186050415, 0.65083909034729, -0.9468206167221069, -0.6001918315887451, 1.60446035861969, 0.421286016702652, 0.4089601933956146, 0.40335577726364136, 1.8682276010513306, 0.4386291205883026, -1.0293077230453491, -1.0452886819839478, -0.7533079385757446, 2.4912707805633545, 0.9674222469329834, -0.4103565216064453, -1.2755072116851807, -0.7491098046302795, -0.2845141589641571, 0.1079685166478157, 0.1999536007642746, -0.5703862309455872, 0.0771414265036583, -1.0373798608779907, -0.48071035742759705, -1.2123761177062988, -0.43393656611442566, -0.6214084625244141, -0.5487681031227112, 1.2033812999725342, 0.5234420895576477, 0.6485452055931091, 0.3287648558616638, 1.671348214149475, -0.3576481342315674, 0.7033805251121521, 0.5768378973007202, 0.1340089738368988, 0.15057753026485443, -1.4445269107818604, 1.2559670209884644, 2.903491735458374, 0.47888094186782837, -0.2765217423439026, -1.2517579793930054, 2.2552690505981445, 1.2073355913162231, -0.3453463613986969, 1.529402256011963, -0.33642613887786865, -0.6903754472732544, 1.0186406373977661, 1.537923812866211, -0.09327894449234009, -0.5120529532432556, -0.7423936128616333, 1.1170434951782227, 0.03771233931183815, -0.9346361756324768, 1.6404236555099487, -1.1327171325683594, -1.6503033638000488, -1.1021513938903809, -0.6085623502731323, 0.32117998600006104, -0.5455093383789062, -1.3968507051467896, 0.09220618009567261, 0.9683723449707031, 1.7183337211608887, 0.1973503679037094, 1.8438267707824707, 0.040884315967559814, -1.9631245136260986, 0.041105397045612335, -0.22408263385295868, 0.6598954200744629, 1.3469338417053223, -0.3298937976360321], [0.5492904186248779, -0.7934813499450684, 0.8663407564163208, 0.39002519845962524, -0.6888181567192078, -0.5757762789726257, -0.6032971739768982, 1.8554012775421143, -0.8205080628395081, -0.2222982496023178, 0.2519562244415283, -1.9494038820266724, -1.6601245403289795, 1.3344523906707764, -0.60368412733078, 0.27996206283569336, -0.06312218308448792, 2.5515685081481934, 1.0076874494552612, 0.09514518082141876, 0.8900032043457031, 1.284271478652954, 0.536358654499054, -1.6022274494171143, -1.680594801902771, -0.6317634582519531, 0.3804241120815277, 1.3911691904067993, -0.5381540060043335, -1.132228970527649, 0.4241371750831604, 0.265956312417984, 0.4631134569644928, 0.21037374436855316, 1.7176713943481445, 0.5257721543312073, -0.8626726269721985, -1.494261622428894, -0.10268035531044006, -1.0924007892608643, 0.2069319188594818, -1.6052076816558838, -0.13349959254264832, 0.9426639080047607, 0.017932403832674026, 0.9466490745544434, -0.5942659974098206, -0.7666924595832825, 0.09712129086256027, 0.3078746199607849, -1.3984644412994385, -0.3923301696777344, 0.14180819690227509, 0.3833523392677307, -1.1561367511749268, 0.5300725102424622, 0.028236234560608864, 0.5442771911621094, -0.4827767610549927, -0.9576355218887329, 0.8006616830825806, -0.9237334132194519, 0.62374347448349, -1.2975959777832031, -0.18631954491138458, 1.0157464742660522, 0.10992055386304855, -2.117243528366089, 1.3792333602905273, -0.6631077527999878, -0.0048398845829069614, -0.5284528136253357, 0.10741081833839417, -0.4502618610858917, 0.39738011360168457, 1.4612665176391602, 0.24495922029018402, -1.6583638191223145, -1.1677677631378174, -0.06616737693548203, -1.1271722316741943, 0.4774037301540375, -2.8251307010650635, 0.42789897322654724, -0.7341794371604919, -0.9734293818473816, 1.97171151638031, 0.9204210042953491, -1.0402098894119263, 1.6656020879745483, 1.2069547176361084, -1.380278468132019, -0.2087346762418747, -0.08291787654161453, -0.043309878557920456, -0.9739804267883301, -1.4077719449996948, -0.040953803807497025, 0.6834862232208252, -0.15793383121490479, 0.5667532086372375, 0.6897429823875427, 1.0192384719848633, 0.6732936501502991, -1.2812117338180542, -1.5850683450698853, -1.0098026990890503, 0.6184379458427429, -1.6488693952560425, -0.5565758943557739, -0.8202366232872009, -0.3732476234436035, 0.6145372986793518, 1.4626799821853638, -0.5576091408729553, 1.5845365524291992, 0.5437525510787964, -0.5707741379737854, -1.610092043876648, 0.4424714148044586, -0.7034303545951843, -0.19128936529159546, 0.1731709986925125, -0.38564950227737427, 0.11081410944461823, -0.16882435977458954, 0.3043248653411865, -0.7656900882720947], [0.02758481167256832, 0.2148396074771881, 0.07288787513971329, -0.9127931594848633, 1.9709341526031494, -0.8570075631141663, -3.3839848041534424, 0.6964888572692871, -0.934733510017395, 1.542377233505249, 0.05389128997921944, 0.7431786060333252, 0.6660935878753662, -1.3761324882507324, 0.03895297273993492, -0.13026268780231476, -1.2623043060302734, 2.0424182415008545, -0.8440884947776794, -1.4263639450073242, 0.7362164258956909, 1.503461480140686, 1.5580735206604004, 0.5235539078712463, -1.7125965356826782, -0.07205090671777725, -0.6282748579978943, -0.584479808807373, -0.13422593474388123, -1.8069674968719482, 0.5089566111564636, -1.5380946397781372, -0.6260656118392944, 1.490670084953308, 1.293088674545288, 0.5117726922035217, -0.6905786395072937, 2.0260608196258545, 0.8006495237350464, -1.1236492395401, 0.9915874600410461, 0.3310115933418274, 0.7841216921806335, -0.15736794471740723, 0.5826088190078735, 0.34031444787979126, -0.5323930382728577, -1.065563440322876, 0.11059215664863586, 1.518836498260498, -0.5442919135093689, 0.6164247989654541, -1.4973920583724976, -0.09670208394527435, 1.8181978464126587, 2.1003293991088867, -0.32839131355285645, -0.6078751087188721, 0.43959903717041016, 1.6412162780761719, 0.6008890867233276, 1.0151995420455933, -0.6966211795806885, -0.0854206457734108, -0.6646965146064758, -1.7354809045791626, 0.4855835735797882, -0.36422088742256165, -0.5594303011894226, 0.004002981353551149, -0.3877575397491455, -0.6778295040130615, -0.4031832218170166, 0.34006914496421814, 0.62713223695755, 0.279251366853714, -0.9234368801116943, 0.6548978686332703, -0.14518198370933533, -0.2416745275259018, -0.8223685622215271, 0.15671798586845398, 0.36249300837516785, -1.2894086837768555, 1.5183494091033936, -0.9987421035766602, -0.4691011905670166, 0.30854952335357666, -0.5843976140022278, -1.6919264793395996, -2.05224347114563, 0.3093855082988739, -0.5582367181777954, 0.8362981677055359, 1.082542061805725, 0.045520808547735214, -1.116193413734436, -1.0515480041503906, -1.6484310626983643, 0.8244467973709106, 0.23496897518634796, -0.611035168170929, 1.8447620868682861, 1.3443653583526611, -0.5073627233505249, 1.6214776039123535, 0.40251022577285767, -1.06549072265625, -0.2953190207481384, -0.39553359150886536, -0.4677228629589081, -0.03317795693874359, -0.05068294331431389, -0.6268194913864136, 1.3707512617111206, 1.6690523624420166, -1.542679786682129, -0.10155239701271057, -0.058088503777980804, 0.26985445618629456, -1.5966427326202393, -0.7754957675933838, 0.656367838382721, -0.4544074535369873, 0.8723391890525818, -0.9294787645339966, 1.9886964559555054, -0.8283749222755432], [-1.5298362970352173, 1.009442687034607, 0.9524197578430176, -0.609268069267273, -2.2577006816864014, 1.6041969060897827, -0.6767407655715942, 0.20020602643489838, -1.1764122247695923, -0.5279743075370789, 0.353133887052536, -0.7725990414619446, 0.9549540877342224, 0.4043748378753662, -0.10308258980512619, -1.8630539178848267, -0.09712066501379013, -0.3916275203227997, -1.0878616571426392, 1.0127886533737183, -1.9647597074508667, -0.7622861266136169, -0.8539232611656189, 1.1838994026184082, -0.9345760345458984, 0.7651927471160889, -0.37515491247177124, -1.2206037044525146, 2.310729503631592, -0.6915726065635681, -0.9397198557853699, 0.40230804681777954, -0.373053640127182, 0.5873727798461914, -0.2613333463668823, 0.7965426445007324, -1.3561162948608398, -2.1681745052337646, -0.4518383741378784, -0.1661255806684494, 0.5754691958427429, 0.3878859281539917, -1.1017850637435913, 0.16144497692584991, 1.2365063428878784, -0.10767384618520737, 1.1435848474502563, -1.2688579559326172, 0.8785127401351929, 1.30637526512146, 1.0250194072723389, -0.5968165397644043, 0.762134850025177, 0.32449355721473694, -0.6661900877952576, -0.043152984231710434, -0.8956700563430786, -1.2780920267105103, -0.1249890848994255, 1.0225614309310913, -0.5135709047317505, 1.271405577659607, -0.09444929659366608, 0.5489454865455627, 0.4680451452732086, 1.2663756608963013, -0.09650915861129761, -0.9854211211204529, -1.2848458290100098, 0.7232807874679565, -0.9441748857498169, -0.7861281633377075, -1.5751551389694214, -0.47598281502723694, -1.2444829940795898, -0.4586172103881836, 0.6980727910995483, 0.882262647151947, 1.184733510017395, 1.0164791345596313, 0.6296395659446716, -0.43084946274757385, 3.346230983734131, -0.7568998336791992, -0.6234286427497864, 0.11605000495910645, 0.19592367112636566, 1.3910492658615112, -0.512730598449707, 1.2302007675170898, -0.6906545162200928, -0.8783622980117798, -1.0965367555618286, -2.025907278060913, -0.13998469710350037, -1.552544355392456, -0.0159518551081419, -0.7435780167579651, 0.22888846695423126, 0.7231948375701904, 0.6857719421386719, -1.1430484056472778, -0.5568716526031494, 1.037034273147583, -0.0520659014582634, -0.5023939609527588, 0.6273663640022278, 0.5799707770347595, -0.10364396870136261, 0.23115696012973785, 0.20209096372127533, -0.0481293611228466, 0.266635000705719, 0.2633019983768463, 0.15045809745788574, 0.07662805914878845, -0.768445611000061, 0.8765341639518738, 0.1484232395887375, -1.3496302366256714, -1.3004950284957886, -1.5191972255706787, 0.9926650524139404, -0.26111701130867004, 1.0568997859954834, -0.05073792114853859, -0.42285770177841187, 0.45612120628356934], [0.6396188735961914, -0.42963898181915283, -1.085890531539917, -0.481680303812027, 0.3480980098247528, 0.32618939876556396, 0.4605219066143036, -1.2298322916030884, 0.15753191709518433, -1.3466554880142212, 0.825116753578186, 0.22394534945487976, -0.17626236379146576, -0.44798392057418823, -1.12446928024292, -0.9533511400222778, -0.24319246411323547, 0.4471683204174042, 0.30966562032699585, -1.5006784200668335, 0.6909112930297852, 0.47545337677001953, -0.6374252438545227, -1.032960057258606, 0.16802483797073364, 2.052281379699707, -0.9993624687194824, -0.024241480976343155, -0.14918822050094604, 0.9741335511207581, 0.7950755953788757, -0.1211291030049324, -0.18445204198360443, 1.559016466140747, 0.6828076243400574, 0.5306652784347534, -0.2896920442581177, 0.5268494486808777, 0.35018450021743774, 0.5608544945716858, -0.10674510896205902, -0.44823628664016724, -0.05779971554875374, 0.7478899955749512, -0.23067502677440643, 0.15368318557739258, -1.336710810661316, 0.271857351064682, -0.06521426886320114, 1.373846411705017, -0.4263056516647339, 0.7801476120948792, 0.3585864007472992, -0.37572017312049866, -1.3108073472976685, -1.1294801235198975, 0.5390967726707458, 0.9450000524520874, -0.02222595363855362, -0.6488404870033264, 0.1670527309179306, -0.5809874534606934, -0.37802088260650635, 0.4000273048877716, -0.23499982059001923, 0.8776863217353821, 0.08975869417190552, -0.8318321108818054, -1.2209417819976807, 1.6567456722259521, 0.7544595003128052, -0.6814244985580444, -0.04264746233820915, -0.5875461101531982, 0.4722743630409241, -0.2830774784088135, 1.2293545007705688, 1.0495356321334839, 0.6579853296279907, 0.537284791469574, -1.5234956741333008, -0.18472889065742493, -0.4213477671146393, -0.35747653245925903, -0.7943642139434814, 0.5552070140838623, 0.8135530948638916, -0.25952836871147156, 1.5640653371810913, 0.35889488458633423, -0.09596216678619385, 0.3503522276878357, -0.15787222981452942, 0.11593764275312424, 0.42224881052970886, -1.0442728996276855, 1.3218580484390259, 0.5453274846076965, -1.1344029903411865, -0.2768356502056122, 2.0513417720794678, 1.9014184474945068, -0.8754458427429199, 0.12388355284929276, -1.2139290571212769, -0.766314685344696, 0.24348624050617218, -0.5876122713088989, 0.1677943468093872, -1.8995248079299927, 1.0444046258926392, 1.684037208557129, 2.4010376930236816, 0.22790510952472687, 0.5274595022201538, 0.7063463926315308, 1.2894785404205322, -0.5649385452270508, -0.45559072494506836, -0.5664196610450745, -0.6776303648948669, 0.810187041759491, -0.31676921248435974, 0.020215261727571487, 1.4692965745925903, 0.861644983291626, 0.7201904058456421, 0.968985378742218], [-1.78742516040802, -0.23470938205718994, 0.4367963969707489, -1.1436278820037842, -0.1949075609445572, 1.4872989654541016, -0.8872644305229187, -0.6733525395393372, 1.5060827732086182, -0.9529366493225098, 1.2239878177642822, -1.559812307357788, -0.15342696011066437, -0.22692547738552094, -0.0644451454281807, 0.6496615409851074, -0.21202954649925232, 0.5549537539482117, -2.0016863346099854, 1.542999505996704, 0.1719866693019867, -0.8959117531776428, -1.1301751136779785, 0.5847247838973999, -1.817754864692688, -1.5236623287200928, 1.060708999633789, -0.16940419375896454, 0.7706347107887268, 0.06237049028277397, 0.3538842797279358, 0.6187134385108948, 1.0303655862808228, -1.7166575193405151, 1.8434425592422485, 0.8000326752662659, 0.9933235049247742, 2.4448516368865967, -0.7218142747879028, 1.3009002208709717, 0.5578927993774414, 0.6646672487258911, 0.2029067724943161, -0.6413428783416748, -0.08767463266849518, -0.9654606580734253, 1.2076143026351929, -0.7396605610847473, -0.5585584044456482, -0.674197256565094, -1.161850094795227, -0.5827890038490295, 0.38304999470710754, -0.09324712306261063, 0.6008724570274353, -0.43212392926216125, 1.0671846866607666, 1.558323621749878, 0.3887435495853424, -0.26132550835609436, 0.22634148597717285, -0.26091235876083374, -1.5674843788146973, -1.720828652381897, -0.2308085560798645, -1.7743867635726929, -0.9254441857337952, -0.7600926160812378, -0.39518842101097107, 1.3379791975021362, 1.060379981994629, -1.6063512563705444, -0.7110645771026611, -0.4037911593914032, -0.8317460417747498, 0.0725339725613594, -2.756913185119629, 0.18300314247608185, 0.44094017148017883, 0.5212404131889343, -1.4370895624160767, 0.19187642633914948, 0.029987197369337082, -0.39981159567832947, -1.00594961643219, -2.4039218425750732, 0.0018172255950048566, -1.9869384765625, 1.5077074766159058, 0.6935285925865173, 0.5003361105918884, 0.4791480004787445, 0.7398269772529602, -1.180895447731018, 0.03614022955298424, -0.6816936135292053, 1.4365088939666748, -1.2673990726470947, -0.7799620032310486, 0.3610839545726776, -0.515679121017456, -0.2757035493850708, 0.9271023273468018, -0.307255357503891, 0.2968699634075165, -0.6255141496658325, -1.2772570848464966, -0.26672816276550293, 2.4068424701690674, -0.8517418503761292, -1.133163571357727, 0.003219055477529764, 0.2333465963602066, -0.484027624130249, -0.5324606895446777, -1.1766752004623413, -0.3212079703807831, -0.05737810581922531, -0.6812156438827515, 0.010861585848033428, -0.16924896836280823, 1.780617356300354, -1.318698763847351, 1.8948160409927368, 0.41636890172958374, -0.43055006861686707, -0.09145709872245789, -1.86543607711792], [1.5059453248977661, 0.039868369698524475, -0.8665314316749573, -0.12262915819883347, -1.1438928842544556, -0.8552184104919434, -0.9025365710258484, -0.7028882503509521, -1.4282786846160889, 0.30604052543640137, -0.3450319766998291, 2.4889516830444336, -0.7773941159248352, -0.6738408803939819, -1.213482141494751, -0.6050442457199097, 0.4640214443206787, 0.866920530796051, -0.9533358216285706, 0.6770533323287964, 0.37185853719711304, -0.13300208747386932, 0.2922287881374359, 0.027112461626529694, 0.6683028340339661, 0.6501584649085999, -0.2513916492462158, 0.33522719144821167, 1.1002193689346313, 0.4974689483642578, 0.9856283068656921, 0.4036930501461029, -1.0515997409820557, 2.0771284103393555, -1.8476570844650269, 1.3798586130142212, -0.597511887550354, -0.8596517443656921, -0.03907882422208786, -0.788718581199646, 0.16136161983013153, -0.381520539522171, 0.5254825353622437, -0.1122363731265068, 1.3480156660079956, 0.9505837559700012, -0.1305207759141922, 2.1116080284118652, -1.0064635276794434, 0.5977064371109009, -1.9108248949050903, 0.659932553768158, -0.9661815166473389, -0.09567391127347946, 0.31155839562416077, -0.33715376257896423, -1.400099277496338, -0.4884832203388214, 0.1654074639081955, -2.354094982147217, -0.36492621898651123, -0.8634868264198303, -0.8681281805038452, -2.1289443969726562, 0.31032028794288635, -1.1864497661590576, -0.36481261253356934, 1.4558191299438477, -0.3965704143047333, 0.8180150389671326, 0.424242228269577, -0.5335540175437927, 0.11257299035787582, 0.5267482399940491, -1.9650849103927612, -1.937766671180725, -0.22162096202373505, 0.956577479839325, 0.004395271185785532, 0.742620050907135, 1.4164916276931763, -0.5598992109298706, 0.1741897314786911, -0.6266056299209595, 0.1310749053955078, -1.1495925188064575, 1.8523530960083008, 0.35426434874534607, -1.4807963371276855, 0.924457311630249, -0.5679517388343811, -0.40081775188446045, -1.204192042350769, 0.48819398880004883, 0.6491689682006836, -0.3445475995540619, 0.5393393039703369, 0.4132494330406189, -0.33444568514823914, 0.9880531430244446, 1.375691533088684, 1.1262660026550293, 0.19932924211025238, -0.5275534987449646, -0.08668737858533859, -0.13222502171993256, -0.8243232369422913, 1.0408730506896973, -2.4406909942626953, 0.5753656625747681, 1.3665891885757446, -1.2149244546890259, -0.7648742198944092, 1.3558708429336548, 0.7598811984062195, -0.6402469873428345, -1.770959496498108, -0.9977108836174011, 0.5755562782287598, -1.7233803272247314, 0.009407952427864075, 0.1787707805633545, -0.4312717616558075, -2.305595874786377, -0.1709422916173935, 0.28220823407173157, -1.057071566581726, -0.3191598355770111], [0.7896985411643982, -0.08162444829940796, 0.9454213976860046, -1.5515038967132568, 2.162351608276367, -0.9102132320404053, -0.14549900591373444, 1.1141248941421509, -1.2111029624938965, 0.2571883499622345, 1.3793504238128662, -0.7199987769126892, 0.30451586842536926, 0.8371661901473999, -1.2995208501815796, -1.2234148979187012, -0.5141778588294983, -0.2542330026626587, 0.308948278427124, 0.14853303134441376, -0.7409741282463074, 0.6769601106643677, -0.6218489408493042, -0.18399593234062195, -0.2672884166240692, -1.1132243871688843, 0.411973237991333, -1.0019451379776, -1.631604790687561, -1.353651762008667, 0.57417893409729, -0.5426466464996338, 0.5095959305763245, -0.4597151577472687, 0.8775494694709778, 0.7224586606025696, -0.3840128779411316, 0.8581092357635498, 0.3271197974681854, -0.5580608248710632, -1.0482497215270996, 0.07894671708345413, 0.9028684496879578, 0.7087423801422119, 0.5263793468475342, -0.12935571372509003, -0.4081045985221863, -0.22293387353420258, -1.4822989702224731, -0.9298322200775146, 1.2685496807098389, 1.3604098558425903, 0.8399062752723694, 1.1555414199829102, -1.2812830209732056, 0.8796479105949402, -0.12261554598808289, 1.5726613998413086, -0.7290248274803162, 0.7851109504699707, 0.3820783793926239, 0.6816566586494446, -0.018449651077389717, -0.6877314448356628, -0.2787967026233673, -0.8313582539558411, 0.8792135119438171, 0.08944275975227356, 0.954407274723053, -0.36989715695381165, 0.2276267111301422, -0.1872863918542862, -1.5263601541519165, -1.5734105110168457, 1.0233319997787476, 3.1863858699798584, -1.3601776361465454, 0.023235663771629333, 0.32743093371391296, 1.5852185487747192, 2.116384983062744, 0.38776686787605286, 1.6296253204345703, 0.15379014611244202, -0.5541000962257385, -1.2924034595489502, -0.6109440922737122, -0.6859108209609985, -0.8390633463859558, 1.1869617700576782, 1.5194957256317139, -1.572033166885376, -0.812411367893219, -1.7918603420257568, -0.8847068548202515, 0.9969789981842041, -0.014594310894608498, -0.4605356454849243, 2.3584158420562744, -0.9551058411598206, 1.0264806747436523, 1.3769069910049438, -0.8044055104255676, 0.8086786866188049, 1.7269761562347412, 0.3762187659740448, 0.23945721983909607, 1.1654199361801147, 1.0909768342971802, -1.3636798858642578, 1.0081562995910645, 0.6759835481643677, 0.12937010824680328, -0.049227677285671234, 1.3411067724227905, -0.9386646151542664, 1.6988624334335327, 1.0114233493804932, -0.12801826000213623, -1.3727936744689941, -1.3258767127990723, -0.20486465096473694, -0.15431800484657288, -0.5271247029304504, 0.036941248923540115, -0.9930267333984375, 0.8358432054519653, -1.014023780822754], [0.729263186454773, -0.0016983902314677835, 0.8065717816352844, -0.18338528275489807, -0.3625558912754059, -0.883499264717102, -0.7161645293235779, -0.3619851768016815, 0.8201950788497925, 1.6558647155761719, 1.2373323440551758, 0.2566966712474823, -1.406363844871521, -1.337231159210205, -0.6995762586593628, -0.7471082210540771, 0.2553918957710266, -0.4785539209842682, -1.2512685060501099, 0.6356698870658875, 1.3020564317703247, 1.1148386001586914, 1.8363957405090332, 1.169622540473938, -0.9858670830726624, 0.35471853613853455, 1.148103952407837, 0.609975278377533, 1.4488182067871094, -0.0311888474971056, -1.3473131656646729, -0.7784854173660278, 1.253343939781189, 0.2060365080833435, -0.5026068091392517, 1.5019985437393188, 0.6711028218269348, 0.002594341989606619, -0.3921181261539459, -0.5742980241775513, -0.33492475748062134, -2.0492751598358154, -0.7184592485427856, -1.2124645709991455, 0.7383723258972168, 0.14023829996585846, -0.1969347894191742, 0.10929175466299057, 0.9984004497528076, 0.16485989093780518, -1.5035243034362793, 0.3762781620025635, 0.6837145090103149, 0.6555584073066711, 1.9546167850494385, 2.3605122566223145, -1.5971646308898926, 0.5967397093772888, -0.6441986560821533, 0.2096739411354065, 1.108336329460144, -1.034043550491333, -2.535015344619751, -1.5337966680526733, -0.2079869955778122, -0.9716155529022217, -1.1737127304077148, 1.402526617050171, -1.3268375396728516, 1.7243951559066772, -0.26208072900772095, 0.6485638618469238, 0.5695523619651794, -0.3353867828845978, -0.04589123651385307, -0.5617597103118896, 1.8328043222427368, -0.3791161775588989, -0.7401520013809204, 1.409869909286499, 0.734869658946991, 0.2888696789741516, -0.32124975323677063, -0.2384178191423416, -0.07045429944992065, -0.3583816885948181, -0.8804309964179993, 1.3264636993408203, 0.5456516742706299, 1.1796128749847412, 0.7400645017623901, 0.24881581962108612, -0.6897000670433044, 0.34242796897888184, 0.4257976710796356, -0.29578378796577454, 0.08967756479978561, 0.6569899916648865, -0.9615516662597656, -0.6227359175682068, -1.1991822719573975, 0.013421501964330673, -0.5062472820281982, 0.8465662002563477, -0.937239944934845, 0.8794440031051636, -1.9990031719207764, 0.5692781805992126, -0.2558612823486328, 0.9275676608085632, -0.5128101706504822, -0.7927919626235962, 1.3965493440628052, 0.18978402018547058, -1.5396571159362793, 0.4762628674507141, -0.4685068726539612, 0.746422529220581, 1.2875632047653198, -1.387963056564331, 0.284572958946228, 1.5710221529006958, 0.4334150552749634, 0.12410955876111984, 0.5564892888069153, 0.006351541727781296, -0.01136485580354929, 1.4170095920562744], [-0.06749938428401947, -0.4783070683479309, -0.6257086396217346, -1.2042243480682373, -1.2027708292007446, -0.527532696723938, 1.1724900007247925, -1.615982174873352, 0.24894727766513824, 1.4422316551208496, 0.4545348882675171, 0.7591071128845215, 2.8934619426727295, 1.860055923461914, -1.7415480613708496, -0.396594375371933, -0.9344667792320251, -0.24702313542366028, 1.3805456161499023, -1.7897286415100098, -0.1848922073841095, 0.5591942667961121, -0.07799845933914185, -0.7439844012260437, 1.086154580116272, -0.8925167322158813, 0.20800678431987762, -0.14606541395187378, 2.19326114654541, 0.2917481064796448, 0.30624741315841675, 1.7054580450057983, 0.09407338500022888, -0.17278902232646942, -1.6979050636291504, -0.7581697106361389, 0.7571828961372375, -0.43567514419555664, -1.9472265243530273, -0.378018319606781, 0.14072559773921967, 0.9540719985961914, 1.5627470016479492, -0.012130506336688995, -1.4372520446777344, -2.0630388259887695, 0.7188668847084045, -0.03098302148282528, 0.7170552015304565, -1.7367758750915527, 0.7869026064872742, 0.8907056450843811, -0.7019484639167786, 1.901402473449707, 0.9314496517181396, 2.3905107975006104, 0.5260595083236694, -0.5157217979431152, 0.034717295318841934, -0.3499525785446167, -0.23182162642478943, 0.4935588538646698, 0.6815164685249329, 0.6005849838256836, -1.6547269821166992, -0.24095064401626587, 0.3554975092411041, -1.4212148189544678, -0.5174816250801086, -0.61541748046875, -0.4883207082748413, 0.781671404838562, 0.736031711101532, 0.33700692653656006, 1.4205914735794067, 0.1612202227115631, 0.7242408394813538, 1.129501223564148, -0.748478353023529, 0.7569625973701477, 0.5064489841461182, 0.30150556564331055, 0.021887371316552162, 0.7174223065376282, -0.08284314721822739, -0.08023486286401749, -1.5157475471496582, -0.7668848633766174, 0.5776873230934143, -0.2741281986236572, 0.567535400390625, 0.7098727226257324, 0.5086765289306641, 0.7698103189468384, -0.2204742282629013, -0.5805498957633972, -0.9385620951652527, -0.032828494906425476, -0.9127072691917419, -0.8254047632217407, 0.8484931588172913, 1.0040397644042969, -0.06267672032117844, 0.557529628276825, -1.1376689672470093, 0.862786054611206, -0.8746211528778076, 0.7605767250061035, 0.47607576847076416, 1.050297737121582, -1.4202643632888794, -0.32471075654029846, -0.38579046726226807, 0.44728517532348633, -0.09333071112632751, 1.054598093032837, 0.3164886236190796, 0.3626829981803894, 0.3153693974018097, 0.10080995410680771, -1.5070136785507202, 1.0082260370254517, -0.6072859168052673, 0.787807285785675, -1.2661521434783936, 1.0619916915893555, -1.7228399515151978, 1.2641818523406982], [1.194467544555664, -0.32449957728385925, -0.698631763458252, -0.00800045020878315, 0.017460277304053307, -0.47595733404159546, 1.37395179271698, -0.4088101089000702, -0.29789894819259644, 1.0406416654586792, 1.2945083379745483, 0.6481435298919678, -0.5011442303657532, 0.1137128695845604, 2.414783239364624, 0.8125231266021729, -1.3741331100463867, -0.406270295381546, -0.11649728566408157, -2.560922384262085, -0.7887495160102844, 0.9211614727973938, 0.5219019651412964, 0.22401714324951172, -0.08129967749118805, 0.6988741159439087, -1.817646861076355, -0.9930893182754517, -0.1813984513282776, 0.6641094088554382, -1.270339012145996, -0.2513595223426819, 0.15050281584262848, 0.409736305475235, 0.9847484827041626, 0.6768029928207397, -0.8678854703903198, 0.5024005174636841, 0.19865034520626068, -0.07776237279176712, 0.7130621671676636, -2.0383920669555664, -0.1537129282951355, -0.9247813820838928, 0.4658443033695221, -2.012641429901123, -0.6255871653556824, -0.11735700070858002, -2.2290091514587402, -0.30163314938545227, 1.0682544708251953, -0.10334967821836472, -0.34142524003982544, -1.1296889781951904, 0.8451225757598877, 0.44704243540763855, -1.1482000350952148, 1.727577805519104, 1.2317934036254883, -0.6602336764335632, 0.1510561853647232, 1.3026230335235596, -0.6268664002418518, 0.7651921510696411, 1.3902673721313477, -0.39931946992874146, -0.7549911737442017, -0.5028098821640015, 0.22196835279464722, 1.216048240661621, 0.6061100959777832, -0.7910786271095276, 0.10813792049884796, 1.812287449836731, -2.5460689067840576, -0.13918817043304443, 0.511612057685852, 1.0807464122772217, 0.15251979231834412, -0.49969372153282166, 0.3570246398448944, 2.584465503692627, -0.27190014719963074, -0.345385879278183, -2.1907384395599365, 1.0076572895050049, -0.32759881019592285, -0.22122685611248016, 0.9063054323196411, 0.9353286623954773, 0.518304169178009, 0.045841947197914124, -0.8868184089660645, 0.40971583127975464, 0.7527781128883362, 1.0211845636367798, 1.274446725845337, -0.44827404618263245, -0.8990836143493652, -2.0089404582977295, -0.24147431552410126, -0.759048342704773, -0.6068650484085083, 1.613957166671753, 1.0185601711273193, 0.10618285834789276, 0.9554566740989685, -0.4042578339576721, 1.3662567138671875, -0.5576445460319519, 0.021477054804563522, -0.002384069375693798, 0.9802230000495911, 1.1429309844970703, 0.40389788150787354, 0.24728280305862427, -0.2477055937051773, -1.026302695274353, 1.5880694389343262, 0.6522030234336853, 1.0387171506881714, -1.230589509010315, 0.19155466556549072, -0.6418771147727966, 0.8909258842468262, -0.6493650078773499, -0.33339351415634155, -1.433189868927002], [0.6188638210296631, -0.21193905174732208, -1.7187392711639404, -0.36206746101379395, -0.40846848487854004, 0.7143807411193848, 0.04750942811369896, 0.6473219990730286, 0.38198256492614746, 1.8675254583358765, 0.9607875347137451, 0.3077095150947571, -0.7636480927467346, -0.0258137546479702, -1.6374638080596924, -1.3563313484191895, 0.920420229434967, 0.19802235066890717, 0.7056212425231934, -1.1695244312286377, 0.12074434012174606, -0.17003168165683746, -0.9841856360435486, 2.0052547454833984, 0.08335884660482407, -0.463495671749115, -1.4186925888061523, -1.1052074432373047, -0.009470540098845959, -0.8194776177406311, 0.7972047328948975, -0.2211792767047882, 0.2365817278623581, 0.06730296462774277, 1.5819882154464722, 0.43830353021621704, -1.6251499652862549, -0.32383307814598083, -1.3559340238571167, -1.0600767135620117, 0.6171680092811584, -0.177061066031456, 1.050520420074463, -0.016710931435227394, -0.9887387752532959, -1.4455009698867798, -1.2871583700180054, -0.9291995763778687, 2.415234327316284, -0.6762409210205078, 0.1528642177581787, -1.3720033168792725, 0.287070095539093, 0.22257430851459503, 0.6203895211219788, -1.4392844438552856, -0.32478612661361694, -1.8299914598464966, 1.0987340211868286, 0.010171152651309967, 0.0710967555642128, -2.073943614959717, 1.8922182321548462, -1.7306674718856812, 2.737656354904175, 0.6469122171401978, -0.6767134666442871, -0.5611374378204346, -0.3323565423488617, -0.17779213190078735, -0.12004005908966064, -0.0961155891418457, -2.0737149715423584, 0.11528578400611877, -0.0864991694688797, -1.033447027206421, 0.37970173358917236, -0.29260534048080444, -0.4541396498680115, 1.2906142473220825, 0.6387073397636414, -1.7562974691390991, 0.12929660081863403, 0.051871176809072495, -0.44332975149154663, -0.11241092532873154, -1.6715211868286133, 1.8068830966949463, -0.9806217551231384, 0.25227198004722595, 0.13658136129379272, 1.9462978839874268, -0.4431852698326111, -0.7679176330566406, -1.3239003419876099, -0.20840784907341003, 0.09338510036468506, -0.4069015383720398, 0.15782178938388824, -0.4269219934940338, 0.13779328763484955, 0.4426524341106415, -0.29995280504226685, -2.51621150970459, 0.0022846602369099855, -1.0056689977645874, 0.4380577504634857, -0.771165668964386, 2.1738152503967285, 0.28433528542518616, -0.7022075057029724, 0.671710193157196, 0.06439340859651566, -0.7664111256599426, 0.6466835737228394, 0.14317865669727325, 0.6564945578575134, 1.0703086853027344, -0.8448012471199036, 0.056208036839962006, -0.18711644411087036, -0.7046138644218445, -0.5537586212158203, 0.6991015076637268, 0.8854724168777466, 1.67750084400177, 1.2355012893676758, 0.49549347162246704], [0.2524837553501129, -0.23826226592063904, -0.06550565361976624, 0.49058154225349426, -1.626918077468872, 0.2367902547121048, 0.015934770926833153, -0.6570637226104736, 0.945297122001648, 0.32315078377723694, -0.627415120601654, 0.8930124044418335, 1.5113097429275513, 0.8623868823051453, -0.1194668635725975, -1.1347219944000244, 0.027248764410614967, 0.4033128619194031, 2.425447940826416, -1.1607396602630615, -1.4336650371551514, -1.7648502588272095, -1.930333137512207, 0.648368239402771, -0.39765632152557373, 0.07909911870956421, 1.5534636974334717, 0.5694276094436646, 0.1971854269504547, 0.17214705049991608, 1.3740899562835693, 0.11851800233125687, 0.4710807502269745, 0.684714138507843, -0.8095592260360718, -0.7288190126419067, -0.007757535204291344, 1.5875152349472046, 0.5208949446678162, -0.515214741230011, -0.9673864841461182, 0.4888467490673065, -2.2689595222473145, 0.29358503222465515, 2.1265149116516113, 0.005800062790513039, -0.7212867140769958, -1.2522519826889038, -0.6747007966041565, 1.7763190269470215, 1.5149656534194946, 0.9333489537239075, 0.3728146255016327, 1.1677377223968506, -0.8236262202262878, -0.00500616617500782, 0.28523197770118713, 0.16738393902778625, 1.085313320159912, 0.485662043094635, -1.4662561416625977, 0.5516749620437622, -0.1038651168346405, -0.1098487600684166, 0.16007208824157715, 0.07089924067258835, 0.2357284128665924, -0.6418081521987915, 0.30356940627098083, -0.8259989619255066, 0.760009229183197, 1.1465145349502563, 1.6069507598876953, -0.39942169189453125, 0.3496149182319641, -0.5236749649047852, -1.6210402250289917, 0.7215144038200378, 0.10975655913352966, -1.865721583366394, -0.5523295998573303, -0.7078616619110107, -1.1291435956954956, -0.8163885474205017, 0.529038667678833, -0.8046649694442749, -0.39097073674201965, 1.3492313623428345, -1.1431483030319214, -0.8262571692466736, -0.3766688406467438, 1.2737735509872437, 0.19607728719711304, 1.6013535261154175, -0.4872523546218872, 0.9105564951896667, -1.5228655338287354, 0.2481182962656021, 0.4993892312049866, -0.9970421195030212, 0.14171986281871796, 0.11116795986890793, -1.348188877105713, -0.14731420576572418, 1.461421251296997, 0.5738410353660583, -0.2540540099143982, 1.137833833694458, -0.15686264634132385, -0.34730595350265503, 1.2375036478042603, -1.1901861429214478, -1.6759611368179321, 0.793156087398529, 0.169210284948349, 0.15326109528541565, 0.5037152767181396, -1.4776220321655273, -3.3275671005249023, -1.2076126337051392, -0.8085033893585205, 0.32404625415802, 0.594182550907135, -0.1735095977783203, -2.2528016567230225, 2.4550483226776123, 0.7582440376281738, -1.9873093366622925], [-1.762427806854248, -1.192084789276123, 2.680372017493937e-05, 0.08185455203056335, 0.4405253231525421, -0.6218486428260803, -0.38881218433380127, -0.23362407088279724, 0.7053989171981812, 0.045762334018945694, 0.6728762984275818, 0.16057394444942474, -0.9326840043067932, 0.029815834015607834, 0.7220005393028259, -0.896753191947937, 1.8094960451126099, 0.44700655341148376, -0.532911479473114, -0.6468725204467773, 0.9135497808456421, 0.622460663318634, -1.0613316297531128, -0.23394125699996948, -1.1526737213134766, -0.9062275290489197, -0.7116936445236206, 1.536436915397644, 1.4599475860595703, 0.0037021988537162542, 0.3667736351490021, -0.7076192498207092, -2.0256381034851074, -0.6631011962890625, 0.12467911094427109, 1.5210648775100708, 1.5057399272918701, -1.5079188346862793, 0.19028182327747345, -0.10631892085075378, -0.12034282833337784, 1.0457813739776611, 1.0567573308944702, 2.295271396636963, 1.472930908203125, 1.1173218488693237, 0.46271222829818726, 1.1322299242019653, 1.466863989830017, 0.7845969200134277, 0.3059394955635071, -0.12627016007900238, 0.4636193811893463, 1.8466546535491943, -0.5736827850341797, -0.9788786172866821, -0.3914147913455963, 0.8607218265533447, 0.09751478582620621, -0.19208259880542755, 3.4437873363494873, 1.1278908252716064, 0.03930902108550072, 0.8800099492073059, 0.5570074319839478, -0.4148787260055542, -0.5795914530754089, 1.1640039682388306, -1.3843811750411987, -0.10219504684209824, -0.42171189188957214, -1.009526014328003, 1.2853915691375732, -0.9290066957473755, -0.014569913037121296, 0.27870309352874756, 0.10316066443920135, 0.10552489757537842, 0.19240812957286835, -0.07902427017688751, 0.6053169965744019, 0.9647715091705322, 0.9965614080429077, 0.3939811885356903, 0.3834225535392761, -0.2187509387731552, -0.25608280301094055, -0.714895486831665, -0.650497317314148, 3.2277870178222656, -0.4562032222747803, 0.5796638131141663, -0.7643862366676331, 0.8882933259010315, 1.2273616790771484, 0.22141921520233154, -0.6918537020683289, 0.14939424395561218, -0.3970486521720886, 2.4006145000457764, 0.8602530360221863, 0.25269827246665955, -1.4127848148345947, 0.04204656183719635, -1.1816637516021729, -1.2695828676223755, -0.6655547618865967, -0.0006450089858844876, 0.03934358060359955, -0.2959049642086029, -1.168488621711731, 0.42832425236701965, 1.0277135372161865, 1.3598154783248901, -0.7983847856521606, -0.3165707588195801, -1.0366623401641846, 0.5360389947891235, 2.293604850769043, -0.08188894391059875, 1.3529095649719238, -0.28831005096435547, 0.609937846660614, 0.873595118522644, 0.2663496136665344, 1.218404769897461, -2.6864240169525146, -0.6575366854667664], [1.0206507444381714, -0.8343442678451538, -0.4666362702846527, 0.4882301688194275, -0.44258394837379456, 0.8734792470932007, 0.3610915243625641, -1.1691778898239136, -0.18611933290958405, -2.5632262229919434, 0.35799118876457214, 0.8204365968704224, 0.5026938915252686, 1.6731271743774414, 0.7641749978065491, -2.638198137283325, -0.044152501970529556, 2.0473201274871826, -0.8326857089996338, -1.1608924865722656, 0.74752277135849, -0.40298110246658325, -0.03814222291111946, 0.5634633302688599, -1.2221671342849731, 0.6550551056861877, 0.632561206817627, 1.0095006227493286, -1.1251857280731201, 0.1289198249578476, -0.9111809730529785, -0.5687307715415955, 0.8697311282157898, -1.3578630685806274, 1.0964491367340088, 1.3075686693191528, 2.024728536605835, -1.1862549781799316, -0.32614341378211975, -0.7549206614494324, -1.1325595378875732, -0.9754641056060791, -1.0027198791503906, -0.6523907780647278, -0.41934388875961304, -0.44766128063201904, -0.8346887826919556, 0.9367237091064453, -0.493758887052536, 0.0852983295917511, 0.979150652885437, 0.4350954592227936, 1.2480227947235107, 1.104533076286316, 0.7836874723434448, -0.7037001252174377, -1.5103988647460938, 0.329257607460022, 0.030684098601341248, 1.0725629329681396, 0.45518332719802856, -0.45810455083847046, -1.654098629951477, -0.4758945107460022, -0.05210462957620621, 0.2949216663837433, 1.6741660833358765, -1.5282270908355713, -0.33605527877807617, -0.13097412884235382, 0.7931026816368103, 0.6596057415008545, 0.6879376173019409, -0.4393327832221985, -0.6718267202377319, 0.9786757826805115, -1.167933464050293, -0.45763495564460754, 0.7385006546974182, -1.4966249465942383, 0.8805961608886719, -0.6326669454574585, 0.2196875959634781, 0.2392529845237732, -1.2274824380874634, 0.9023923277854919, -0.6811454892158508, 0.685851514339447, 1.5881257057189941, 0.44929614663124084, 0.9465717077255249, 0.8802775740623474, -0.6906211972236633, -0.06276340037584305, -1.303790807723999, 0.3775325417518616, -0.857417643070221, 0.14944539964199066, 0.6304967999458313, 0.9424740076065063, 0.5905694365501404, -1.317159652709961, 2.275902509689331, 0.4414190351963043, 0.1931839883327484, 0.45484867691993713, 1.5725723505020142, 0.36901381611824036, 0.07853888720273972, 0.42637619376182556, 0.12946492433547974, 2.079148054122925, -0.9751450419425964, -1.3328930139541626, -0.3627348244190216, -0.41446781158447266, 0.31569623947143555, -1.0395256280899048, 0.022124020382761955, -0.4895446300506592, -0.3673481345176697, 1.1884517669677734, -0.6716864705085754, -0.28715500235557556, -1.7839421033859253, 1.167210340499878, 0.7336692810058594, 1.7037303447723389], [-1.265405535697937, 0.4287044405937195, 0.5604130029678345, -0.821962833404541, -0.24104250967502594, -1.7742117643356323, -1.0911973714828491, -0.07335972040891647, 1.500464677810669, -0.3861238956451416, 0.04791128635406494, 0.7053163051605225, 1.077221393585205, 1.1234800815582275, 0.3756142258644104, 1.8495261669158936, -0.8120260834693909, 2.543185234069824, -0.3807889223098755, -0.06625350564718246, 0.11837580054998398, -1.8462001085281372, 0.804619312286377, -0.12306543439626694, -0.07095279544591904, -1.2465403079986572, 1.6177388429641724, 0.19424374401569366, -1.1129153966903687, -1.580125331878662, -0.306155800819397, 0.028411246836185455, 0.6287596225738525, 0.5647454857826233, 0.5532609224319458, 0.395906001329422, -2.0329842567443848, -0.8322077989578247, -0.23090910911560059, -0.7853487730026245, 0.9905074834823608, 1.551626205444336, -0.8896405696868896, 1.5179970264434814, -0.5325135588645935, 0.38833069801330566, -0.06654204428195953, 1.8075991868972778, 0.5602331161499023, 0.25776147842407227, 0.5073128342628479, 0.9547812342643738, -1.246209740638733, 0.5849860906600952, 0.29306501150131226, -0.6127350330352783, 1.3735069036483765, 0.6274141669273376, -0.29508116841316223, -0.5495039820671082, -1.431458830833435, 0.5651988387107849, 0.5134323239326477, 0.9625048041343689, -0.11850842088460922, -0.337457537651062, -1.352662205696106, 0.27416664361953735, -0.27822884917259216, 1.6311184167861938, 1.1406439542770386, 0.5000449419021606, -0.22881042957305908, -2.134340286254883, 0.46670976281166077, -0.22449640929698944, 1.556934118270874, -1.0140855312347412, 0.5512295365333557, -1.2686387300491333, -1.8818572759628296, -0.032593365758657455, -0.806751549243927, -0.3697383403778076, -1.2275464534759521, -1.5540648698806763, 1.1813737154006958, 0.47123172879219055, -1.2545477151870728, -0.21646782755851746, -1.2298582792282104, 1.4415024518966675, -0.6489186882972717, 1.18610680103302, -0.6989073157310486, 0.4432486593723297, 0.02136640064418316, 0.9166767001152039, 1.186211109161377, 0.24277834594249725, 1.2693145275115967, 0.38654008507728577, 1.2219350337982178, 0.2533469498157501, 0.8169081807136536, -0.12537731230258942, 0.4839057922363281, -0.2759853005409241, -0.2409900575876236, -2.3239054679870605, -0.8835363984107971, -0.07819323986768723, 1.6608608961105347, -2.734461784362793, -0.4792209565639496, 0.029176272451877594, -0.03738196939229965, 0.4505784809589386, -0.7571143507957458, -1.412224292755127, 0.22433936595916748, 0.46983712911605835, -0.8353843092918396, 0.7389252781867981, -0.2668114900588989, 0.5584309697151184, -0.8904204368591309, -0.5385475754737854], [-0.9311976432800293, -0.010653261095285416, -0.7570925951004028, 0.5785310864448547, 0.0225654449313879, 0.5517839193344116, 0.7239413857460022, 0.2014581561088562, -0.23061928153038025, -0.8600326776504517, 0.05015813186764717, 1.3539464473724365, -0.48082321882247925, 1.547558307647705, -0.180273175239563, -0.4414171278476715, -0.10191989690065384, -0.4630071520805359, 0.02270621806383133, -0.8941230773925781, -0.7964659333229065, -0.12029992789030075, 0.9640699625015259, -3.129502296447754, -2.0964877605438232, -1.4350111484527588, 1.3302204608917236, -0.07360175251960754, -0.7852350473403931, 0.789596676826477, 0.26986798644065857, -0.07664252072572708, -0.4856586158275604, -0.5590471029281616, 0.4561501741409302, 1.1876304149627686, -0.40676596760749817, -0.676462709903717, 0.5708749890327454, 1.646284580230713, -0.26596599817276, 0.6685656309127808, 0.00545937055721879, -0.2975444197654724, 0.08794207870960236, 0.02597324550151825, 0.13802272081375122, -0.5655051469802856, -1.1909422874450684, -1.2261799573898315, 0.9797372817993164, -0.2741956412792206, 0.746959924697876, 1.3416534662246704, -0.8794901371002197, 1.0840338468551636, 1.2554315328598022, 0.3000902831554413, 0.15957069396972656, -0.47705498337745667, 1.439723253250122, 1.4036661386489868, -1.2679933309555054, 1.1420494318008423, 0.12360832095146179, -0.47280988097190857, 0.25068846344947815, 0.7974170446395874, 1.0482277870178223, -0.7332401871681213, 1.3992135524749756, 0.46910253167152405, 0.6881349086761475, 1.1486518383026123, 0.560928225517273, -0.6919224262237549, -0.8997833728790283, -2.008913993835449, -1.7372008562088013, -0.6948908567428589, -0.34068602323532104, 1.3814828395843506, 1.2891894578933716, 0.6774585843086243, 0.03963441401720047, 0.9536728858947754, 0.6818540692329407, 0.618452787399292, -0.6504807472229004, -0.5915144085884094, 1.2516415119171143, 0.7686544060707092, 0.18530704081058502, -0.21571780741214752, 0.23948846757411957, 0.9235127568244934, -0.8635061979293823, -0.4678865671157837, 0.39695659279823303, 0.5465933680534363, -0.8592319488525391, 0.8911420702934265, 0.055219948291778564, -2.370953321456909, -0.011612463742494583, 2.2731850147247314, 1.302925705909729, 0.6660824418067932, 0.6706814765930176, -1.0900721549987793, -0.07159296423196793, -0.9887717366218567, -0.6360064148902893, -0.12304026633501053, 0.5730677247047424, 0.41924944519996643, 0.6342911720275879, 0.44500014185905457, 0.6804412603378296, 0.4250911474227905, -2.0562899112701416, 0.5020449161529541, -1.163745641708374, 0.27106478810310364, -1.1367087364196777, -1.2306692600250244, -1.120657205581665, -0.7509300112724304]]], [[[1.6608656644821167, -0.9929745197296143, -1.4664926528930664, 1.5593538284301758, 0.323598176240921, 0.757665753364563, 0.9351444244384766, -0.10911166667938232, 1.2762407064437866, 0.3569048047065735, -0.02128351479768753, 0.7363632321357727, 2.1075379848480225, -0.2432316541671753, -1.310320258140564, 1.2507199048995972, 0.9651563167572021, -0.9707432389259338, 1.1378215551376343, 0.12056328356266022, 0.46392884850502014, 1.411973237991333, 0.004591512493789196, 1.731569766998291, 0.4194035530090332, -0.46256008744239807, -1.9245142936706543, -1.4973946809768677, -0.3957246243953705, 1.9847126007080078, -1.3726612329483032, 0.9682257175445557, -0.15939831733703613, -1.6612317562103271, -0.12155012786388397, 0.13128015398979187, -0.8224217295646667, -0.5087643265724182, 1.2461973428726196, -2.312772512435913, -0.2083948850631714, -2.531740188598633, -2.150578737258911, -1.045764446258545, -0.004839468747377396, 1.1545219421386719, -0.5290187001228333, 0.742729902267456, 2.11346435546875, 0.6960972547531128, -0.4127572476863861, 0.44406867027282715, 0.031002739444375038, 0.011607306078076363, 1.1542807817459106, -1.4386756420135498, -1.154376745223999, 0.8245162963867188, 0.9304804801940918, 0.6826049089431763, -1.8081326484680176, -0.5775821208953857, -0.025307707488536835, -0.08238117396831512, -1.0728086233139038, 1.7500219345092773, 0.34211933612823486, -0.02345825545489788, -0.5187267065048218, 1.248716115951538, -1.5216227769851685, 0.19622504711151123, -0.2808699607849121, 1.126732349395752, 0.23175044357776642, 1.1441781520843506, 1.6685329675674438, 0.3177569508552551, -1.2700101137161255, -0.3545156717300415, -1.1634658575057983, -0.7355013489723206, 0.6993885636329651, -0.07230640947818756, 0.9456909894943237, -0.7560347318649292, -1.2189031839370728, 1.2628540992736816, -1.2726542949676514, 1.335913062095642, -1.6124646663665771, 1.6375329494476318, 2.836073160171509, 0.24912308156490326, -0.9724614024162292, -0.9617464542388916, -1.4984928369522095, -1.7763915061950684, -1.5103925466537476, -0.044464435428380966, 0.23253333568572998, -0.33025407791137695, -0.8035099506378174, -0.3087083399295807, 0.2541203200817108, 1.1691255569458008, -0.8786315321922302, 1.3742446899414062, 0.5533982515335083, -2.154409170150757, -0.45522186160087585, -0.3390860855579376, 1.1054835319519043, -0.8274555206298828, 0.16107313334941864, -0.32272636890411377, -1.051092505455017, 0.1127023696899414, -0.723507285118103, -1.2409087419509888, 0.8511985540390015, 0.5064923167228699, -0.35715246200561523, 0.1395287662744522, 0.7020984292030334, 1.739052176475525, -1.165784239768982, 0.726015567779541], [-1.9964401721954346, -2.102613687515259, -0.5806875228881836, -0.9263618588447571, -0.6610506772994995, 0.05767769739031792, 0.06491675227880478, 0.5999616384506226, 0.032972563058137894, 0.7766328454017639, 1.4897191524505615, -0.1506425142288208, -0.694260835647583, -0.7318558096885681, -0.882606565952301, -0.003549721324816346, 1.211919903755188, 0.4418219029903412, 0.2794354557991028, -0.6136555671691895, 1.9241523742675781, -0.20130415260791779, 1.3728467226028442, 1.074897050857544, -0.830913245677948, 0.8463963866233826, 0.5345578193664551, 0.3633800745010376, 0.87093186378479, 1.1358089447021484, 0.054645899683237076, 0.03333840146660805, -1.3271987438201904, -0.7948142290115356, 0.29215267300605774, -0.6557037234306335, 0.21437177062034607, 1.3473055362701416, 1.0751841068267822, -1.2700915336608887, -0.6293333172798157, 0.7428802847862244, -0.32127645611763, 0.4263059198856354, -1.167319416999817, 1.0324734449386597, 1.1434857845306396, 1.0607925653457642, -0.1975180208683014, -0.7319194078445435, -0.03337676450610161, -2.373286247253418, 1.2610384225845337, 0.8754037022590637, -0.030399002134799957, 2.107816696166992, 0.8136743307113647, 1.0629267692565918, -0.0629933625459671, 0.3486992418766022, 1.6197625398635864, 1.3469158411026, 0.38682788610458374, 0.2673190236091614, -1.3052281141281128, -0.06004280596971512, 1.5912210941314697, 1.757978081703186, -1.448807954788208, -0.013991937041282654, 0.14758343994617462, -0.790715754032135, 0.8143147826194763, 0.5694617629051208, 1.1343491077423096, 0.15684626996517181, -0.3711917996406555, 1.008001446723938, 0.6580455303192139, -0.407978892326355, -0.7474397420883179, 0.05719431862235069, -1.072521686553955, 0.3713490664958954, -0.6068120002746582, 0.005547694396227598, -1.912523627281189, -0.11666914075613022, -0.7520172595977783, -0.6137784123420715, 0.5423376560211182, 1.022599697113037, 1.0444300174713135, -0.1271769255399704, -0.028162222355604172, -0.657602846622467, -0.16233950853347778, 0.6026577353477478, 0.8383723497390747, 1.4840139150619507, -0.32044124603271484, -0.5197470188140869, 0.21108444035053253, -0.22925087809562683, -0.5789276957511902, -1.4468170404434204, -1.456221342086792, 0.9396435022354126, -0.45113199949264526, -1.0969547033309937, -2.893402576446533, 0.2612963020801544, 1.5098720788955688, -0.3452506363391876, -0.8752186298370361, -0.09370003640651703, -0.9830496311187744, -1.3803547620773315, -1.0791995525360107, 0.03987952694296837, -1.5916653871536255, 0.5268546938896179, 0.2856534719467163, 0.2835196554660797, -0.28825676441192627, -0.8319708108901978, -0.4969594478607178, 1.2685812711715698], [-0.21425947546958923, 0.04743330925703049, 0.04183292016386986, -0.9521557688713074, -0.2549390196800232, 0.2640702724456787, -0.24515432119369507, 2.7085301876068115, -0.7172317504882812, 0.0863603875041008, 1.5584471225738525, 1.0964680910110474, -1.1289174556732178, 0.26004815101623535, 0.07578536868095398, 1.727597951889038, 0.3986194431781769, -0.23885244131088257, 1.1143220663070679, -0.743988573551178, 0.14447292685508728, -0.9306420087814331, -1.1180222034454346, -0.9510906338691711, -0.4014800488948822, -1.629534125328064, -1.4128077030181885, 1.06931734085083, 0.6250808238983154, 1.2099295854568481, 0.6858812570571899, 1.1849840879440308, 1.6503355503082275, -1.3740466833114624, -1.3238645792007446, 0.7332442402839661, 0.5069093704223633, -0.12420342862606049, 0.3489607274532318, -0.6666639447212219, -1.4242033958435059, -0.6643447279930115, -0.2374565303325653, 1.1377874612808228, -1.1111996173858643, -0.8245950937271118, -0.577235758304596, -0.8728966116905212, -0.18584981560707092, 1.14798903465271, -0.007883480750024319, 0.3873642086982727, 0.2364063858985901, 1.3551119565963745, 1.2356783151626587, -1.4658328294754028, 0.4515629708766937, -0.2796967327594757, 0.37501561641693115, 1.0917223691940308, -0.18115931749343872, -1.1488875150680542, 1.2097023725509644, -0.08742481470108032, 0.5099914073944092, -1.3383513689041138, -1.3962405920028687, -0.4668503999710083, 0.07751885056495667, 1.706024169921875, -1.2570642232894897, 1.5139487981796265, -0.4989013075828552, 0.5843784213066101, -0.006072235759347677, 0.7935488820075989, -0.9904910922050476, 0.43041834235191345, 1.1224088668823242, -1.8753197193145752, -0.8261761665344238, -0.8801229596138, -0.8069900870323181, 0.24560290575027466, -0.1020481139421463, -0.23527707159519196, 1.362221598625183, -1.8614193201065063, 1.0188952684402466, -0.38077765703201294, 1.3751821517944336, 0.34871289134025574, -1.1513272523880005, 1.961682677268982, -0.4960632026195526, 2.2546563148498535, -0.024079402908682823, 1.2682411670684814, -0.16729941964149475, -0.4267549514770508, 0.7643177509307861, -0.18017350137233734, 0.15919093787670135, -0.7109624743461609, -0.9690541625022888, 0.4368363916873932, 0.22078144550323486, 0.37795573472976685, -0.4079159200191498, -0.8279685378074646, -2.120908260345459, 0.28649064898490906, 0.26290780305862427, -0.25249752402305603, -0.13653624057769775, -0.5440382957458496, -0.30464744567871094, -0.2337121218442917, -0.025857893750071526, -2.1334593296051025, -0.11507615447044373, -1.122147560119629, 0.5497798323631287, -0.196129709482193, 0.6313197016716003, -1.7358348369598389, -1.493788480758667, -1.6267731189727783], [-0.5505497455596924, -0.19811129570007324, -1.2503803968429565, 1.970686912536621, -0.34703347086906433, -0.7005438804626465, 0.7811746001243591, -2.078188180923462, 0.917831301689148, -1.4009722471237183, -0.6386042833328247, -2.046955108642578, 1.6922107934951782, -0.8548984527587891, 0.7363781332969666, -2.5749285221099854, 0.3938335180282593, 0.8748101592063904, -1.5322303771972656, 2.2999932765960693, -0.2676171660423279, 1.069042444229126, 1.2298487424850464, 0.353139728307724, -1.3670815229415894, -0.43541523814201355, 1.2749042510986328, -0.7197731733322144, 2.4390084743499756, -1.489762544631958, -0.3429661691188812, 1.6275484561920166, -1.1054552793502808, 1.0096174478530884, 0.3875340521335602, 0.8096805810928345, -0.5026800036430359, 1.4568090438842773, 0.9503998756408691, 1.6294820308685303, -0.2234473079442978, 0.032149266451597214, 0.013810363598167896, -0.4363330900669098, -0.38910236954689026, 0.16126656532287598, -0.011536865495145321, -0.23545575141906738, 0.48381486535072327, -0.37926918268203735, -0.150893896818161, 2.2877988815307617, 0.28481703996658325, -0.18002161383628845, 0.9319416284561157, -1.245549201965332, 0.5322572588920593, -0.1984012871980667, -1.2136963605880737, -0.28181931376457214, 0.7899413108825684, 0.0013946564868092537, -0.5923289656639099, -0.4014548361301422, 0.5223828554153442, 0.7784428596496582, 0.8156693577766418, -0.9041280150413513, 0.6158870458602905, 0.10728075355291367, -0.7560591697692871, 1.2041072845458984, -0.5837329626083374, -1.0430355072021484, -0.9302859306335449, 0.08848828822374344, -1.586658000946045, 1.9382225275039673, -0.7304549813270569, 0.13548687100410461, -0.6780520081520081, 1.5300674438476562, 1.5028893947601318, -1.1724594831466675, -0.23621906340122223, 1.6747589111328125, -1.8213982582092285, 1.0336859226226807, -0.9314408898353577, -1.0511693954467773, 1.2060340642929077, -0.26308178901672363, -1.665157437324524, -0.7799237370491028, 0.2889349162578583, 0.32908061146736145, 0.4736088812351227, 1.6749053001403809, 1.3448078632354736, 0.44269463419914246, 0.27756166458129883, 0.19386717677116394, -0.36559128761291504, 1.1349494457244873, -0.1106841117143631, 0.5822001695632935, 0.36714082956314087, 0.8001604080200195, -0.7908287048339844, -0.7062769532203674, -0.5635712146759033, -0.5165150761604309, 0.7665253281593323, 0.03231004253029823, -0.9914653897285461, 0.4724433720111847, -1.389217734336853, -0.3901178538799286, -0.0513521246612072, 0.686750054359436, 0.1727239340543747, 0.1268075555562973, 0.5902796387672424, 1.138857126235962, -0.050068601965904236, 0.7197964787483215, 0.6090124249458313, -1.0207374095916748], [-0.42329615354537964, 0.374260812997818, -0.5466390252113342, 1.6564087867736816, -0.16264067590236664, -0.4069341719150543, -0.2781166434288025, 2.0701375007629395, 1.924803614616394, 0.2352054864168167, -2.1134562492370605, 0.7358654141426086, 0.0837399885058403, -0.030112670734524727, 0.5849205255508423, -0.8661463856697083, -0.48526743054389954, 0.3111029267311096, 1.3361244201660156, 0.3293566405773163, 1.0437642335891724, 0.5073984861373901, -0.14203263819217682, -0.17382246255874634, 0.519723117351532, -0.013471085578203201, -0.7186289429664612, -0.14200715720653534, -0.02063458226621151, -0.2088194191455841, 0.417955607175827, -0.9985076189041138, 0.8880451917648315, 1.226322054862976, 0.2782800793647766, -1.6566035747528076, -0.43031179904937744, -0.9696776866912842, -0.36367061734199524, -0.3774700462818146, 0.9819931387901306, 1.2186641693115234, 0.8736507892608643, -0.5311530828475952, -0.2280960977077484, 0.5328569412231445, 0.4413386583328247, -1.5528638362884521, -0.20227952301502228, 0.0795259028673172, -0.46790623664855957, 2.292637348175049, 1.6032639741897583, -2.660183906555176, -1.1051865816116333, 1.821772813796997, -0.8442822098731995, 0.2142883539199829, -0.40737995505332947, -1.2910830974578857, 0.1712268739938736, 0.7816287279129028, 0.2521945536136627, -0.956420361995697, -0.8584909439086914, 0.7925528883934021, 1.240185022354126, -0.8250104188919067, -0.8837298154830933, 0.41938695311546326, -0.09423372894525528, -0.30183565616607666, -0.9396365880966187, 0.915492594242096, -0.9605063796043396, 0.22424915432929993, 1.0694154500961304, 1.627454400062561, 0.16402284801006317, -1.4762765169143677, 0.9433311820030212, 0.7251377701759338, -1.1928293704986572, 0.5103395581245422, -1.8415634632110596, 0.03808846324682236, 0.33540695905685425, -1.95828115940094, -0.08071566373109818, 0.5632948875427246, 0.025779830291867256, 1.2021067142486572, -1.9634782075881958, 0.10646059364080429, 0.18975059688091278, 0.7955427765846252, 1.0787428617477417, -0.9484449028968811, -0.6604211330413818, 0.005972529761493206, -0.024437561631202698, 1.003495216369629, -0.4940679669380188, -0.3500893712043762, 0.6085608005523682, -0.817243754863739, 0.310242235660553, 1.8204902410507202, 1.2530757188796997, -0.904984176158905, 1.0266780853271484, -0.5011579990386963, -1.3449394702911377, 0.4492053687572479, 0.4077381491661072, 0.06738133728504181, 0.1362621933221817, 0.3513287603855133, 0.740542471408844, 0.28688183426856995, -1.1841801404953003, -1.4931175708770752, 0.05121924728155136, 0.03158798813819885, -1.1375696659088135, 1.1725728511810303, -0.15155211091041565, 1.4196239709854126], [0.21325312554836273, -0.2265487164258957, -0.8951221704483032, 0.18149760365486145, 2.115088939666748, 1.0226398706436157, 1.7006702423095703, -0.8427842855453491, -0.37697726488113403, 0.7660285830497742, 1.0280184745788574, 1.946890115737915, -0.8587013483047485, -0.638937771320343, 1.2190477848052979, 0.7948975563049316, -0.9943915009498596, -1.0581440925598145, 0.16633601486682892, 1.2668043375015259, 0.3515658378601074, -0.2236211746931076, 0.710072934627533, 1.2356159687042236, 0.027515588328242302, -0.7804701328277588, -0.046119946986436844, 0.51133131980896, -0.8205925822257996, 0.12152097374200821, 0.08669048547744751, 0.24135549366474152, -0.3906519412994385, -0.05820588022470474, -0.562440812587738, 1.4010130167007446, -0.46702954173088074, 1.321550726890564, -0.44516682624816895, 0.14952951669692993, -1.1781460046768188, 0.6660581231117249, -0.4738917350769043, -0.4316524565219879, 1.2451163530349731, 1.4096813201904297, 0.7838598489761353, -0.9156242609024048, -0.989404022693634, -0.24843628704547882, 0.13946989178657532, 0.725165605545044, -0.22134317457675934, -0.8700234889984131, 0.15591329336166382, 0.8569458723068237, -0.2529126703739166, 0.8121943473815918, 0.9183804392814636, 0.44604629278182983, -0.3509660065174103, 0.4238583445549011, 0.44086915254592896, -0.5918261408805847, 2.142254114151001, -1.5980021953582764, -0.32266470789909363, 0.21759121119976044, 0.3654792904853821, -1.1719856262207031, 0.025933794677257538, -0.2854575216770172, 0.2629894018173218, 1.7478817701339722, 1.4255073070526123, 0.4472638666629791, -1.2797534465789795, -2.346665620803833, 0.18827883899211884, 0.019658681005239487, -1.151668906211853, 0.28725236654281616, -1.3149542808532715, 0.1391899585723877, -1.5626435279846191, -0.5551157593727112, 1.4744371175765991, 0.41547614336013794, -0.2172292321920395, -0.3576202392578125, -1.2623823881149292, 0.8107835650444031, 0.6202383041381836, -0.4756668210029602, -0.09669719636440277, 1.0498249530792236, 0.3196662962436676, 0.6893523335456848, -0.5378246903419495, -0.8362372517585754, 0.053116798400878906, -0.7685422897338867, -0.3959483504295349, -0.740850567817688, 0.34626343846321106, -0.3190392553806305, -1.7655636072158813, 2.0745129585266113, 0.023957904428243637, -0.4008105993270874, 0.4034699499607086, -1.6238148212432861, -0.07468023151159286, 0.8293211460113525, 0.6020545363426208, -0.5018529891967773, 1.4828306436538696, -0.46727582812309265, -1.0306295156478882, -1.383837103843689, 1.0790728330612183, 1.3638397455215454, 1.4226313829421997, -0.6180285215377808, 1.6262081861495972, -0.6931120157241821, -0.536689043045044, -0.21852600574493408], [-1.2084672451019287, 1.0541749000549316, 1.3651665449142456, -2.0696356296539307, 0.5756976008415222, -0.5177742838859558, 0.6408157348632812, -0.7908511161804199, 0.8447362780570984, -0.7069944143295288, -0.7368674278259277, 0.029601242393255234, 0.6090594530105591, 0.30679431557655334, 1.6609987020492554, 1.598730206489563, 0.18595272302627563, -0.0932072177529335, 1.5814826488494873, -1.2140213251113892, -1.012337327003479, 0.3551267087459564, -0.9317677617073059, -0.423644095659256, -0.9145358204841614, 0.15735264122486115, 0.2002507746219635, -2.275240898132324, -0.14549387991428375, 2.2651584148406982, -1.704550862312317, 0.8804214000701904, 0.4839012026786804, -1.0924162864685059, -0.8971792459487915, -1.2025526762008667, 1.1738667488098145, 0.37889304757118225, -0.21303488314151764, 0.48240959644317627, 0.34642133116722107, -0.7262177467346191, -0.9334153532981873, -0.7177596688270569, 0.2671647071838379, 0.7389867901802063, 1.073197364807129, -1.408681035041809, -1.0188122987747192, -0.4597577154636383, 0.7741724848747253, -0.3652275800704956, 0.07365076243877411, -1.4451203346252441, -0.9994686245918274, 0.0992342159152031, 3.4894139766693115, 0.19461855292320251, -0.8706644773483276, -0.8052988052368164, 0.9604918360710144, -1.338229775428772, -0.02010454051196575, -0.9928267002105713, 2.1243324279785156, -0.005385149735957384, -0.9802179336547852, -1.9932650327682495, 2.694922924041748, -0.2701070308685303, 2.4632620811462402, -0.4778269827365875, -0.15151186287403107, 0.293975293636322, 0.011296534910798073, 1.9517401456832886, -0.6230517625808716, 0.25407490134239197, 0.11877437680959702, -0.10430622845888138, 1.5134817361831665, -0.8501845002174377, 0.22943614423274994, -1.4847517013549805, -0.2845275104045868, -0.5925238728523254, 2.1018893718719482, -0.4658720791339874, -0.09868551790714264, -0.11169172078371048, -1.6586109399795532, -0.9076537489891052, -0.1494247168302536, 0.30224236845970154, 2.732966423034668, -0.13055285811424255, 1.6305445432662964, 0.1982666403055191, 0.23139254748821259, 0.8742870092391968, 1.4441559314727783, 1.2738527059555054, -1.0766146183013916, -1.0735377073287964, -0.29810482263565063, 0.008125335909426212, 1.3808653354644775, -1.6902947425842285, -1.5253868103027344, -1.9529902935028076, 0.7053653597831726, -0.4959488809108734, -0.10225196927785873, 0.5127333402633667, 0.18421940505504608, 1.2294895648956299, 0.6113128662109375, 1.5897150039672852, 0.0415123887360096, -0.21238583326339722, -0.6175408363342285, -0.08225081115961075, -1.1115788221359253, 0.8631501793861389, -0.11252758651971817, -0.6462022662162781, 1.2721353769302368, 1.6543360948562622], [1.022757649421692, -2.2485408782958984, -0.061324797570705414, 0.04033883661031723, -0.6012592315673828, -0.34097936749458313, -1.7580856084823608, -2.350588083267212, 0.3894687294960022, -0.6289887428283691, -0.12479566782712936, 2.2543222904205322, -0.30246442556381226, 1.0687662363052368, -0.152569979429245, -0.18886114656925201, -0.23376010358333588, -0.13251014053821564, 0.398053914308548, -0.4570693075656891, 0.6686216592788696, 1.3027976751327515, -0.28847798705101013, -1.5340723991394043, 0.6923918128013611, 0.2687605917453766, 0.3643264174461365, 0.8648049831390381, -0.39405906200408936, 1.0306800603866577, -0.8014458417892456, 0.0013223804999142885, -0.19449687004089355, -1.1380184888839722, -0.01857464201748371, 0.05714303255081177, -0.6422630548477173, -0.25453421473503113, -1.158976435661316, -1.0445538759231567, -0.02118220552802086, -2.0568480491638184, 0.7654914259910583, 0.34307870268821716, -0.21676260232925415, -0.22810685634613037, 0.7639815807342529, -1.244537353515625, -0.08552223443984985, 0.11674462258815765, -0.05822204425930977, 0.5022276639938354, -1.2048274278640747, -0.24159491062164307, 2.6175942420959473, -0.9872788190841675, 1.0551433563232422, 0.40472373366355896, 0.7994751334190369, 0.7173520922660828, -0.10824253410100937, 0.7081350088119507, -0.7638024091720581, -0.5341587662696838, 1.6476397514343262, -1.0677096843719482, -0.9935702681541443, 0.37024787068367004, 1.3605356216430664, 1.771144151687622, 0.956719160079956, 0.46976929903030396, 1.0967836380004883, 1.2428722381591797, 0.2160635143518448, 1.5374175310134888, 1.0644471645355225, 0.0878627598285675, 0.6563916802406311, -0.501787543296814, 0.24789230525493622, 1.4956002235412598, -1.8932912349700928, -0.6238957643508911, -0.2061724066734314, -0.585587203502655, -0.9033013582229614, -0.46489763259887695, -1.9105452299118042, 0.17764565348625183, -1.5147494077682495, 0.9029554724693298, -0.5225750207901001, 0.9421055316925049, -0.40683698654174805, -0.04963771253824234, -1.1446563005447388, 2.446859836578369, -1.2747926712036133, -1.2485313415527344, 1.1595114469528198, 2.1812584400177, -0.4541035592556, -0.4300854206085205, 0.46964746713638306, -0.7943868637084961, 1.7907469272613525, 1.327099323272705, -0.4583558142185211, 1.6002519130706787, -1.9904661178588867, 0.3867639899253845, 0.01669127121567726, 0.4112633168697357, 0.49095648527145386, -0.7039434909820557, 1.201362133026123, -0.0585351288318634, -0.3655329644680023, 1.0881221294403076, -0.6653012037277222, 0.7524541616439819, 1.299232006072998, 0.4018813967704773, -0.085985466837883, 0.4014609456062317, -2.286958932876587, 0.8663009405136108], [0.0994717925786972, 0.12991583347320557, 1.6925324201583862, 0.42592954635620117, -0.11450473219156265, -0.7598435282707214, -3.028596878051758, -0.2809191048145294, -1.7216169834136963, -0.6715723872184753, -0.35963600873947144, 0.5090265274047852, 0.3344125747680664, 0.9019045829772949, 0.6557528972625732, -0.8967524170875549, -0.5098535418510437, -0.25196731090545654, -0.35193386673927307, -1.1453872919082642, -1.0942983627319336, 0.9641091823577881, -0.738054096698761, 0.5357745289802551, 0.2962540090084076, 0.5631028413772583, -1.2623950242996216, -1.4992376565933228, 0.4722885191440582, 0.2638764977455139, 2.2236075401306152, 0.1267232596874237, 0.45699572563171387, 0.5051693916320801, -0.7657449245452881, -0.0755622535943985, -2.812758445739746, -1.3845726251602173, 0.7062432169914246, -0.15713322162628174, 1.0564771890640259, 0.011784477159380913, 1.441954493522644, -2.576490879058838, 0.782556414604187, 2.0309364795684814, 0.6901929974555969, 0.6921488046646118, 0.0028782107401639223, -1.1485612392425537, 0.652212917804718, 1.1906648874282837, 0.4352533519268036, -0.2048037201166153, -1.5772087574005127, -0.6890736818313599, -1.032578945159912, 0.27287763357162476, -1.4621617794036865, -0.4606151580810547, 1.254097580909729, 1.2916845083236694, 0.8921424746513367, 1.4753766059875488, 1.8143049478530884, 0.8671904802322388, 1.3864188194274902, 0.046202968806028366, 2.436171770095825, -0.9175273776054382, 0.7261912226676941, -0.1676904410123825, 0.6081558465957642, -2.4557042121887207, 1.5569778680801392, -0.8993818759918213, 1.7048577070236206, -0.40154024958610535, 0.41865605115890503, -2.2871320247650146, -0.08426282554864883, 0.4638581871986389, -0.022318346425890923, -2.262563943862915, -1.9619550704956055, 1.8197516202926636, 0.08110889792442322, 0.41396796703338623, 1.3123869895935059, -0.2946809232234955, -0.3299158215522766, -0.39627009630203247, -1.9153637886047363, -0.5903897285461426, 1.3985531330108643, -0.877813994884491, -1.3732329607009888, -0.5431124567985535, 0.6512207388877869, 0.896221399307251, -0.8703998327255249, 0.3366457521915436, -0.3789118230342865, 0.946882963180542, -0.5021702647209167, 1.0013625621795654, -1.0985300540924072, 1.459303617477417, -0.6722028851509094, 1.0064777135849, -0.8265585899353027, 1.2456300258636475, -0.7070431709289551, -0.2623065710067749, -0.818359911441803, -1.2435134649276733, -0.9638063311576843, 0.5855305790901184, 0.8150810599327087, -0.7183953523635864, 0.13251321017742157, -1.068956732749939, 2.061819553375244, 0.23914214968681335, 0.33677157759666443, 0.012062237598001957, -0.6343773603439331, 0.3005934953689575], [0.036507006734609604, 0.16433782875537872, -0.6977784633636475, -0.7931747436523438, -1.4726282358169556, 2.261883020401001, -0.4331367611885071, -0.9883637428283691, -1.1932157278060913, -0.45378100872039795, -1.731978178024292, -0.1342535763978958, -3.4076943397521973, 1.439501166343689, 0.7818018794059753, -1.4371260404586792, 1.449208378791809, -1.8149150609970093, 0.3883627653121948, -0.14656320214271545, 0.7552084922790527, 1.0031286478042603, -0.599719762802124, 1.7827562093734741, -1.0401784181594849, -0.26715293526649475, -0.28521308302879333, 0.5906788110733032, -1.1762909889221191, 0.5704147219657898, -0.8031165599822998, -1.5368846654891968, 0.9240491390228271, 1.73393976688385, -1.6735681295394897, 0.7314565777778625, 1.042913794517517, 1.2604351043701172, -0.5842434167861938, -1.2417984008789062, 1.0283282995224, 0.6391215920448303, 0.45960402488708496, 0.024658391252160072, -0.5590133666992188, 0.7243709564208984, -0.6563981771469116, 0.7045620679855347, 0.1261843740940094, 0.6748930811882019, 0.932131826877594, 0.05026141181588173, -2.4550211429595947, 0.9571559429168701, -0.5447468161582947, -0.2813907861709595, -0.19610550999641418, -0.07282672822475433, -0.009099706076085567, -0.9450862407684326, 0.3599216938018799, -0.1298905611038208, -1.0744119882583618, 0.6925168633460999, 0.39902257919311523, 1.2893232107162476, -0.2478407919406891, -0.3088158369064331, -0.568922758102417, 1.4011948108673096, 1.7944154739379883, -1.6999605894088745, 0.43965640664100647, -1.8042075634002686, 0.6484729647636414, 2.880981683731079, 0.0066817994229495525, -0.06535545736551285, 0.18445958197116852, -0.11925952136516571, 0.06708072870969772, 0.09057022631168365, 0.23190927505493164, 1.527382731437683, 0.1580367088317871, 0.37270402908325195, -0.08177217096090317, 0.2715899348258972, -0.7798125147819519, -0.3056286573410034, 0.23411646485328674, 1.0310914516448975, -1.4879592657089233, -0.47385600209236145, 0.8040971755981445, 0.32424789667129517, -0.8488737344741821, -0.08205536752939224, -0.33796870708465576, -0.18066145479679108, 1.1996922492980957, -1.4324829578399658, -1.3917289972305298, -0.8986368179321289, -0.33320367336273193, -0.5504295825958252, -0.11778708547353745, 0.6037957072257996, -0.6897650361061096, -1.1361833810806274, -0.4052535891532898, -1.2294672727584839, 1.5229562520980835, -0.08462761342525482, 0.29669007658958435, 1.533360481262207, -0.6179018020629883, 0.8838189840316772, -0.2672063708305359, -1.9907851219177246, 0.2842390537261963, -0.2665040194988251, 1.4568812847137451, -0.21332690119743347, -0.4039982259273529, 0.3198189437389374, 2.4455220699310303, 0.9726549983024597], [0.5474442839622498, -1.7027907371520996, 1.2150269746780396, 1.3060178756713867, -0.7378864884376526, -0.8369831442832947, -0.7488533854484558, 0.19480006396770477, 0.6317927837371826, 0.10694040358066559, -0.9166567921638489, -1.6426575183868408, -0.7940881252288818, -1.0712090730667114, 0.06527407467365265, 1.3544505834579468, 2.8072993755340576, -1.3430380821228027, -1.4863622188568115, -1.1886786222457886, -1.3404486179351807, 0.25887301564216614, 1.1745059490203857, 0.4329055845737457, -1.7630231380462646, 0.15831436216831207, 0.32801908254623413, 0.31587761640548706, 0.4973222613334656, -2.0106492042541504, -1.7448753118515015, -0.28803110122680664, -0.14166885614395142, -0.2805633246898651, 0.21892684698104858, -2.4763906002044678, -0.8076825737953186, -0.09511295706033707, -0.15891984105110168, -1.342240571975708, -1.1311206817626953, -0.36684712767601013, -1.2229803800582886, -0.7187286615371704, 0.17186357080936432, -0.6638322472572327, 0.44719114899635315, 0.44680142402648926, -0.9666215181350708, -1.5783871412277222, -0.3274833559989929, -0.5409469604492188, -0.8201186656951904, -1.2373263835906982, -0.6597945094108582, 0.8073635101318359, -0.6419575214385986, -0.3511573374271393, 0.45758387446403503, -0.7134923338890076, -0.553678572177887, -1.2576181888580322, 0.8374671339988708, 0.661791980266571, -0.2757124900817871, -0.6827488541603088, 0.431869238615036, 0.8729431629180908, -1.0428235530853271, -3.300081729888916, 0.5608576536178589, 1.371705412864685, -0.175057053565979, -0.032005660235881805, -0.38340359926223755, 0.47215017676353455, -2.201064348220825, -0.16954554617404938, 1.4752322435379028, -1.5464768409729004, -0.2896258234977722, 0.712757408618927, -0.5952954888343811, -2.1673154830932617, 0.3847498297691345, -0.11629974097013474, 0.36704760789871216, -0.4484100043773651, -0.6489123702049255, 0.25737306475639343, 0.8874857425689697, 0.9845581650733948, -1.3466421365737915, -1.4707897901535034, -1.0573883056640625, 0.5434829592704773, -0.5933610796928406, -1.5719380378723145, -0.18262708187103271, -0.57086181640625, 1.04655921459198, -0.45804035663604736, -0.5520979166030884, 0.08727798610925674, 0.5665701627731323, 1.427794098854065, -0.9210771918296814, -0.6922375559806824, -1.4995795488357544, 0.331237256526947, -0.2503642737865448, 1.2840572595596313, -0.6135779023170471, -1.1807366609573364, 0.8651093244552612, 0.9343641400337219, 0.4739750325679779, -2.3606200218200684, -0.35377439856529236, -0.7380313873291016, 0.18677020072937012, -0.783704400062561, -1.2286995649337769, -0.2729666829109192, 0.5599104166030884, 0.3287827670574188, -0.38865047693252563, -1.1826320886611938], [0.5995078682899475, 0.1648818850517273, 0.3218200206756592, 0.1612737476825714, -0.7421958446502686, -0.7038698792457581, 0.4450496435165405, 0.9813508987426758, -2.03399920463562, -1.3960578441619873, 0.9516288042068481, -0.32696110010147095, 0.6220279932022095, -1.154524564743042, -0.26291555166244507, 0.5378345251083374, 0.42926570773124695, 0.6080073118209839, 0.23938409984111786, 0.8174072504043579, -1.2939748764038086, -0.5401527285575867, 1.022939920425415, 1.374459147453308, 0.2568156123161316, -1.1587249040603638, -0.2149958312511444, -0.1815100461244583, 0.3774375021457672, -0.6607845425605774, 1.1646252870559692, -0.18610107898712158, -0.3530912697315216, 0.42036983370780945, 1.2595407962799072, -0.13037748634815216, 0.3761506676673889, -0.6404927372932434, -0.005039147101342678, -1.5638498067855835, -1.3256696462631226, 0.4522448480129242, -0.05860090255737305, 0.09067658334970474, -0.9590337872505188, -2.0655291080474854, 0.7302647829055786, 0.37185540795326233, -0.45221447944641113, 1.182083010673523, -0.534521758556366, -1.1722050905227661, -0.7596557140350342, 0.39352333545684814, -0.2754502594470978, -0.19680854678153992, -0.173846036195755, 0.9580584168434143, -1.0941681861877441, 0.4156127870082855, 0.9996488094329834, 0.18946708738803864, -0.4950171113014221, -0.2966822683811188, -0.26916128396987915, 0.9415962100028992, -0.17319001257419586, -0.2654063403606415, 0.39226609468460083, 0.557803750038147, 1.2139092683792114, 1.079525351524353, -0.007599096745252609, 0.7299219369888306, -0.9297635555267334, -0.34390994906425476, -1.1138365268707275, -0.8809055089950562, 0.5337730646133423, -1.2255789041519165, 0.23405855894088745, 0.03744484484195709, 0.1421358436346054, -1.7883665561676025, -0.884986400604248, -0.7986636161804199, -0.28222206234931946, -0.012858418747782707, -1.2638850212097168, -0.25250986218452454, -0.5081337094306946, 1.456284999847412, 0.15609802305698395, -0.45739591121673584, 0.7370800971984863, 0.5353849530220032, -1.0862752199172974, 0.26009702682495117, -1.2709418535232544, 0.03372307866811752, 0.011808730661869049, -0.5403881072998047, -1.275639533996582, 1.4582054615020752, -0.2683977782726288, 0.5848504304885864, 0.25334861874580383, 1.6423251628875732, -2.3532230854034424, -0.12563759088516235, -1.0289075374603271, -0.9212252497673035, -1.6619513034820557, 1.8903759717941284, 0.1336168795824051, 0.9546483159065247, -0.3428581655025482, -1.716736078262329, 0.872793436050415, 0.3617023527622223, -0.4340190291404724, -0.5189029574394226, -0.682681143283844, 0.048288844525814056, -0.20490801334381104, -0.4513283967971802, -0.4453660249710083, -1.2316250801086426], [2.3461997509002686, 1.0192525386810303, -0.5976314544677734, 1.37975013256073, 1.3037017583847046, -1.1364669799804688, -0.3226187527179718, -0.4092380404472351, 0.1921456754207611, 1.3914039134979248, -0.5695158839225769, -0.06202302500605583, 0.3962736129760742, -0.1270328313112259, 1.169136881828308, 0.41064271330833435, -1.3992749452590942, -2.8574225902557373, -0.8850907683372498, -0.4242760241031647, 0.2001774162054062, 1.2670025825500488, -0.46811941266059875, -1.4202821254730225, 0.9676320552825928, 1.0934031009674072, 0.09144093096256256, -1.8039549589157104, 0.6969355344772339, 0.6867433190345764, 0.2792593240737915, 0.662922739982605, 0.18397679924964905, 0.33681946992874146, 0.21716725826263428, 0.4678262770175934, -0.22594797611236572, 0.2476108968257904, 0.762770414352417, 0.029761048033833504, 0.6077517867088318, -1.9793121814727783, -0.6076171398162842, 0.7437456250190735, -0.39097854495048523, -1.087668776512146, 0.4090336859226227, -1.2294273376464844, 0.1285635232925415, 0.10068203508853912, 1.2067350149154663, 0.58329176902771, 1.5315738916397095, 0.3273860812187195, -0.7669020891189575, 0.8485268354415894, -1.449347972869873, -2.0425758361816406, -0.3241274952888489, 0.18940535187721252, -0.24173910915851593, -0.5807923078536987, -0.07158984988927841, 0.2667216360569, -0.9840983152389526, -1.3448532819747925, 0.9293647408485413, 0.29094398021698, -1.1829087734222412, -0.9666266441345215, 1.4674201011657715, 1.6500871181488037, -1.2447742223739624, -0.47544410824775696, -0.9472091197967529, -0.4510653018951416, 1.3696521520614624, -0.8141410946846008, -0.36051082611083984, 0.09166762232780457, 0.038330115377902985, 0.35393989086151123, -1.8661746978759766, 1.9936131238937378, -0.0350794643163681, -0.013290992006659508, 0.878431499004364, -0.3033393323421478, 0.450369656085968, 1.1914856433868408, -0.6579548120498657, -1.1788444519042969, 0.3859599828720093, 0.822567343711853, -0.06420005857944489, -1.088841199874878, -1.4517518281936646, -0.5610272288322449, 0.8850759267807007, 0.39889028668403625, -0.17200177907943726, 2.2181286811828613, -0.12692482769489288, 0.712270975112915, 1.914811611175537, 1.00777006149292, -0.6578827500343323, -0.417740136384964, 0.7496523261070251, -0.3145521283149719, 0.4499501585960388, 0.9701765775680542, -1.0523427724838257, -0.8021894097328186, -1.2279069423675537, -0.1746026873588562, 2.1556999683380127, 0.28221559524536133, -1.390270471572876, 2.5841026306152344, 1.3580529689788818, 1.5937830209732056, -0.07280457019805908, -0.5292835235595703, -0.18130676448345184, 0.46302908658981323, -0.23225797712802887, -0.2634059488773346], [1.5509133338928223, -0.4088233411312103, -1.1792017221450806, -0.7303470373153687, -0.9568713307380676, -1.3221594095230103, 1.372605562210083, -1.435500144958496, -0.5430478453636169, -1.1104772090911865, 0.9267385005950928, -1.1927385330200195, 0.06007115915417671, 0.5172820687294006, 2.182302236557007, 0.9638048410415649, 0.7100624442100525, 0.6176326274871826, 0.7774367928504944, 0.4472617208957672, 0.3907918334007263, 0.39254069328308105, -1.20896315574646, 0.9459917545318604, -0.9911895394325256, 0.6094961762428284, 0.4057534635066986, 0.6816701889038086, -1.8703858852386475, 0.08586645126342773, -0.2397281676530838, 0.6422147750854492, 0.7106751799583435, 0.6736049652099609, -0.2742319703102112, -0.24517302215099335, 0.5003466606140137, -2.232921838760376, -1.5746822357177734, 0.2504943013191223, 0.29207637906074524, -0.7165873646736145, -0.6380119323730469, 0.871015727519989, -0.16060329973697662, 0.687591016292572, -0.9414569735527039, 0.9349985718727112, -0.12298671156167984, 0.2666158974170685, -0.22170650959014893, 0.015167471952736378, 1.4958477020263672, -0.9509198069572449, -0.10673701763153076, -0.8119942545890808, -0.003184005618095398, -0.5332456827163696, 1.5592067241668701, 0.05152932554483414, -2.682288885116577, -1.2228730916976929, -0.19254006445407867, 0.018834313377738, 1.5373125076293945, 0.1451224535703659, -0.011825411580502987, -0.821588933467865, 0.6131930947303772, -0.6259480714797974, -0.5250438451766968, -0.37446820735931396, -1.0511753559112549, 0.3599584996700287, -0.12286719679832458, 0.02373071201145649, -1.472782850265503, -0.4482850432395935, -0.016322501003742218, -1.843096137046814, -0.14446911215782166, 0.16997657716274261, 0.6293673515319824, -0.21015284955501556, -0.27258676290512085, 0.7389441132545471, 1.8262720108032227, 0.945523738861084, -0.007175952661782503, 1.6789040565490723, -0.6259644627571106, -0.3541829288005829, -0.32252687215805054, -0.6984513998031616, -2.39322829246521, 2.5515477657318115, 0.039179928600788116, 1.0986130237579346, -1.8577075004577637, -0.8926022052764893, 0.23250792920589447, -1.4419535398483276, 1.2120747566223145, 0.15498460829257965, -1.4861027002334595, -0.1434655636548996, -1.0109366178512573, -0.002647290239110589, 0.6658518314361572, 0.8996400833129883, 0.7846612334251404, -0.9066821932792664, 1.290073037147522, 0.11678565293550491, -0.7188636064529419, 0.7685095071792603, 0.21938002109527588, 1.176146388053894, 0.12321033328771591, 0.8759140968322754, 0.905045747756958, -0.5941675901412964, -0.4459609091281891, 0.7356007695198059, 1.7146579027175903, 0.36556223034858704, 2.061692953109741, -1.3934861421585083], [-0.5992990136146545, 1.2748048305511475, 0.5145545601844788, 1.564502239227295, -0.36686795949935913, 0.5043559670448303, -0.7154390811920166, -0.25293272733688354, 0.742655873298645, 0.7083519101142883, 0.4458465576171875, -0.8566543459892273, 0.6765882968902588, 0.3796023726463318, 1.4641246795654297, -1.7315552234649658, 0.1652609407901764, -0.938738226890564, -0.4952794313430786, 2.622936964035034, -0.8064431548118591, -0.9155495762825012, -0.49441981315612793, -1.2615621089935303, 0.5583376884460449, -0.20230333507061005, -0.7620480060577393, 1.5369287729263306, -0.554223895072937, -1.3001508712768555, -0.6135838031768799, -1.636902928352356, -1.07183039188385, -1.6643270254135132, 0.2183224856853485, -0.11821155250072479, 0.1188802719116211, 1.0451819896697998, 0.6926683783531189, -0.3240891396999359, 0.8686485290527344, 0.9950414896011353, 1.681060552597046, -2.04606294631958, -0.488278865814209, 0.06564248353242874, -1.5231949090957642, 0.8948668837547302, -0.6415082812309265, -2.3313112258911133, 0.7419096827507019, -1.4078775644302368, 1.1310642957687378, 2.1381466388702393, -1.6518124341964722, -0.6321909427642822, 0.19227316975593567, 1.7175853252410889, -0.21456179022789001, -0.6863914728164673, 0.9960347414016724, -1.3235325813293457, 0.17402517795562744, 1.535400629043579, 0.9811500310897827, -0.08160707354545593, -0.04023955389857292, 0.8292496204376221, 0.8114244937896729, 1.549248218536377, 0.29394522309303284, -0.1899895966053009, -0.8432779908180237, 0.02785089798271656, 1.9345099925994873, 0.7597634196281433, 0.8252726197242737, -0.09550265967845917, 2.763277769088745, -0.8247602581977844, -0.666990339756012, 2.413231372833252, -0.307574987411499, 1.1816074848175049, -0.31314653158187866, -0.9533395767211914, -1.4800453186035156, -0.3073732554912567, 0.1931874006986618, 0.07532138377428055, -0.08569446951150894, 0.61114501953125, -1.4676473140716553, 0.8799803256988525, 1.2008429765701294, -0.2891683280467987, -1.4481221437454224, -1.451750636100769, 0.11266475915908813, 1.490373134613037, -0.22035138309001923, -0.7258040308952332, 1.3792415857315063, 0.2622178792953491, -0.8933107256889343, -1.6917564868927002, 1.8607367277145386, -0.1429203301668167, 0.32667678594589233, -0.8055523037910461, 0.13969869911670685, -0.2042837291955948, 1.5662328004837036, 1.3562909364700317, -0.49948325753211975, -1.5257645845413208, 0.17131690680980682, -0.26814013719558716, -0.8108484148979187, 1.4547641277313232, 0.23143678903579712, 0.28711220622062683, -1.9799779653549194, 0.9268427491188049, -1.5959298610687256, 0.2740066349506378, 0.18682238459587097, 0.01614036038517952], [-0.36928561329841614, 1.0515828132629395, -1.0523247718811035, 1.4353855848312378, -0.801820695400238, -1.3533737659454346, -1.023819923400879, 1.1719143390655518, -0.11478880792856216, 1.7130898237228394, -0.05405798554420471, 0.8815044164657593, 0.4255404472351074, 0.3875759243965149, -0.14497175812721252, -0.9027343392372131, -1.011674165725708, -0.8718203902244568, 1.433534026145935, -0.41860494017601013, 0.5473690032958984, 0.12221582233905792, -0.18775585293769836, 1.130670428276062, 0.9357638955116272, -0.7111692428588867, -1.086728572845459, -1.0419185161590576, 1.1428453922271729, 0.3930809795856476, 1.154422402381897, 0.23233434557914734, 1.0343835353851318, 0.2452840358018875, 0.10246249288320541, 1.273542046546936, -0.38697630167007446, -0.7060171961784363, 0.9422938227653503, -1.2737653255462646, -0.6203479766845703, -1.1142914295196533, 0.13921289145946503, 0.648474395275116, -1.2379597425460815, 0.21205301582813263, 0.40147310495376587, -0.38523581624031067, 1.1732245683670044, 1.1598891019821167, 3.0642988681793213, 0.30499327182769775, 0.5831247568130493, -0.5189635157585144, -0.2559695541858673, 0.9873115420341492, -0.9933817386627197, 1.0811614990234375, -0.3862614035606384, 0.24680940806865692, 1.693945288658142, -1.2348593473434448, 0.5792966485023499, -0.26024335622787476, 1.021239995956421, -0.7905292510986328, 1.3170456886291504, -0.2550060749053955, -0.2387048453092575, -0.8963600993156433, -1.2589223384857178, 0.40592119097709656, 0.9030691385269165, -0.01553992461413145, 0.45092323422431946, -0.9356256127357483, -0.31862297654151917, -0.17818692326545715, 0.6326696276664734, 1.0290154218673706, 0.16621339321136475, 1.0578324794769287, -0.46864259243011475, 1.1650817394256592, -0.8052837252616882, 0.18980753421783447, 0.3564006984233856, 1.8184503316879272, 0.9354649782180786, -0.7125222086906433, -0.39191755652427673, -0.811100423336029, -0.7532250285148621, -1.7557270526885986, 1.6261003017425537, -1.7359064817428589, -1.0185240507125854, 0.017781037837266922, -0.15457725524902344, 0.7140457034111023, 0.08175109326839447, 2.1220078468322754, 0.35206517577171326, -1.0087007284164429, 0.35752278566360474, 0.7195801138877869, 0.8885539770126343, -0.3103579878807068, -0.2917259633541107, 0.5588807463645935, 0.02878212183713913, -0.09235715121030807, -0.03409045189619064, -2.61413311958313, -0.30210980772972107, 0.15557771921157837, 0.884389340877533, 0.3647312819957733, 0.5621278285980225, -0.2222280651330948, -1.3417690992355347, 0.7816259264945984, -0.55699622631073, 0.06784561276435852, -0.7710034251213074, -0.7064546346664429, -0.05680381879210472, -0.05225365608930588], [0.027288563549518585, -1.3517969846725464, -0.43096885085105896, -0.7046736478805542, -1.7110904455184937, 0.027366647496819496, -0.8341918587684631, -0.7054854035377502, 0.8416050672531128, 0.6439547538757324, 1.2042325735092163, 0.5642094612121582, 1.195348858833313, 0.76617830991745, 0.5864208340644836, 1.007215976715088, 0.7400023937225342, -1.4216039180755615, -0.12177010625600815, 0.2625866234302521, 1.329067349433899, 1.1855977773666382, 0.11180724203586578, -0.8150615692138672, 1.082115888595581, 0.8263551592826843, -0.013431012630462646, -1.025484561920166, 1.8286864757537842, 0.841049313545227, -0.36413198709487915, 0.704055666923523, -3.260796546936035, -1.4218195676803589, 0.9096952080726624, 0.08811472356319427, 1.4164866209030151, -0.10369681566953659, -1.7100163698196411, 0.4348999261856079, 0.4464566111564636, -1.61928129196167, 0.34971901774406433, 0.6624839901924133, 0.39439889788627625, -1.707951545715332, 0.5755072236061096, 0.8367681503295898, -0.33982276916503906, 0.9921531081199646, -1.5335646867752075, -0.5414762496948242, -0.9986068606376648, 0.27809444069862366, 0.4412578344345093, 0.2508820593357086, -0.987720787525177, -0.9017101526260376, 1.3321924209594727, 0.9750851988792419, 0.08839385211467743, 0.277527391910553, -1.7661120891571045, 1.8198844194412231, -0.355784147977829, -2.0523054599761963, 1.147659182548523, 1.508590817451477, 0.7740835547447205, 1.7376965284347534, 0.40112805366516113, 0.3776989281177521, -0.5249802470207214, 0.7371640205383301, 0.004590928554534912, -0.21204403042793274, -1.8122186660766602, 1.7358688116073608, -1.0430972576141357, 0.8434950709342957, -0.08730199187994003, -0.5019582509994507, -0.9077682495117188, -0.3766458332538605, 0.665820837020874, 0.8475690484046936, 0.25467783212661743, -1.1668040752410889, -1.0779623985290527, 0.32314378023147583, -0.5839933156967163, 0.6252511143684387, -0.5681607127189636, 0.08680345863103867, 1.1071009635925293, 1.3662501573562622, -0.2599261701107025, 0.49959906935691833, -0.0331600122153759, -0.22970843315124512, -0.6284240484237671, 1.5853945016860962, -1.6966036558151245, -0.4337395429611206, -0.26081618666648865, 0.5891833305358887, -0.04596941918134689, -1.1122372150421143, 0.9211476445198059, 0.550150454044342, -2.340301752090454, -0.6812240481376648, 0.6982977390289307, 0.7669174075126648, 0.9695095419883728, -2.0026144981384277, 0.7623383402824402, 1.1041797399520874, 0.8987329006195068, 1.4557918310165405, 0.46247678995132446, -1.9010422229766846, 1.3422861099243164, 0.19175827503204346, 0.21006041765213013, 1.2989752292633057, 1.8088250160217285, 1.1685353517532349], [-0.8115513920783997, 1.1398630142211914, 0.6590143442153931, 0.9627275466918945, 0.7894646525382996, 0.3869327902793884, -0.6127420663833618, -0.25298383831977844, -0.32476234436035156, -1.81737220287323, -0.12027871608734131, 0.15956974029541016, -0.1021212488412857, -0.48880550265312195, -0.21803240478038788, -1.2559959888458252, 0.11468201130628586, 0.03933053836226463, -1.3081903457641602, 2.8978161811828613, 2.2609076499938965, -2.136089324951172, 1.162681221961975, -1.172743797302246, 0.9683697819709778, 0.23358562588691711, -1.6431739330291748, 0.05704067274928093, 0.27211958169937134, -1.6802971363067627, 0.8061966300010681, 0.5414358973503113, -0.6352342963218689, 0.6158905625343323, -0.2960226237773895, 0.24478134512901306, -1.9739000797271729, 0.2453240305185318, -0.34907418489456177, -0.6403005719184875, -0.7581571340560913, -1.7917401790618896, 0.16267737746238708, 0.11905614286661148, 0.7611485719680786, -0.6254178285598755, -0.27366286516189575, -0.060464877635240555, -1.0063313245773315, -1.5145537853240967, -1.205536127090454, 1.9067479372024536, -1.1138062477111816, -0.3161613643169403, -0.12135927379131317, -0.015054022893309593, -1.557104229927063, -0.02631049044430256, -0.5037168860435486, -0.14373213052749634, -0.9681218862533569, -0.6534126996994019, 1.5920847654342651, 0.6994050741195679, -0.9033491611480713, 0.8536738157272339, -0.9729039669036865, 0.79845130443573, 0.771740734577179, -0.8788213133811951, 0.17372392117977142, 0.49038976430892944, -0.9615916609764099, -0.3867886960506439, -0.9145900011062622, -0.2700923979282379, -1.4416471719741821, -0.49846985936164856, 0.8813858032226562, 0.8357378840446472, -0.3768608272075653, 0.5686297416687012, -0.616465151309967, -1.0399575233459473, -0.21818400919437408, -1.002959132194519, -0.8059931993484497, 1.0730572938919067, -0.5992192029953003, 0.3316687345504761, 1.1124420166015625, 0.5046401023864746, -0.8459213376045227, 0.3148123025894165, 0.5421496629714966, -0.3134797513484955, -0.06365025788545609, -0.5631455183029175, 0.5793797373771667, -0.11300349980592728, 0.6757500171661377, -2.565046548843384, -0.2731705904006958, -0.23148711025714874, -0.5219901204109192, 0.719258189201355, -0.6050704717636108, -2.0131847858428955, -1.3971002101898193, 0.4253712594509125, -1.2899819612503052, 1.4696533679962158, 0.8137854337692261, 0.6119306087493896, 0.7472707033157349, 0.7746824622154236, -0.2186814248561859, -0.14569123089313507, -0.5956447124481201, 0.22676947712898254, -1.0838209390640259, 0.5932064056396484, -0.3054535686969757, -1.4998079538345337, -2.1553993225097656, -0.5470283031463623, 0.036719564348459244, -0.788121223449707], [0.762209415435791, -1.0543817281723022, -1.4545903205871582, 1.2694666385650635, -0.6677664518356323, 0.22155971825122833, 1.4472109079360962, 0.7755041718482971, 0.6264969110488892, -0.9939369559288025, -0.18811863660812378, -0.41721096634864807, 1.1858984231948853, -1.327888011932373, 0.8550628423690796, 1.0217667818069458, -1.0910485982894897, -0.17359530925750732, -0.5627091526985168, -0.47670209407806396, 2.022282838821411, 0.47360318899154663, -0.33284881711006165, 0.394025981426239, -1.9750579595565796, 0.8206684589385986, -0.06492075324058533, 0.7445167303085327, -0.005128081422299147, 0.8084065914154053, 1.3443163633346558, -1.074934959411621, -0.8067837357521057, 0.3668036162853241, -0.7262880206108093, 0.2756156921386719, -0.2196359932422638, 0.8686690330505371, 1.1035271883010864, 1.1281589269638062, 0.699733316898346, -0.19993922114372253, -0.3731612265110016, 0.6950775980949402, -1.5861304998397827, 0.6313849091529846, -0.4063189625740051, -1.0957118272781372, 1.6076555252075195, -0.9281134009361267, -3.032243013381958, -0.7145140767097473, 1.1612478494644165, 0.29906511306762695, 0.9648363590240479, 0.1871315836906433, -0.06413204222917557, -1.324698805809021, -0.20552533864974976, -0.22520437836647034, -0.03121507354080677, 0.8813943862915039, 1.3298962116241455, 0.7127177715301514, -0.5627895593643188, 0.8158330917358398, -0.5267888903617859, -0.18337981402873993, -0.20690542459487915, -0.5242032408714294, -0.08018998056650162, -0.39401480555534363, -0.6600357294082642, -1.565652847290039, -0.8196731805801392, -0.18396921455860138, 2.5397982597351074, 1.4424158334732056, 0.5583809614181519, -1.8508899211883545, -0.8313151597976685, 0.44746971130371094, -0.8730681538581848, -0.5518978238105774, -0.40257367491722107, -0.4748680591583252, 1.6969901323318481, -0.529161810874939, -2.1029882431030273, 1.5191820859909058, 0.7839827537536621, 0.6913203597068787, 1.2572153806686401, 0.36048224568367004, 1.783178687095642, 0.5677990913391113, -0.4918239116668701, 0.5628944635391235, 0.24967420101165771, 0.26150816679000854, 0.12916849553585052, -0.04146742820739746, -0.342365562915802, -0.9932342171669006, -1.8825825452804565, -0.6519524455070496, 0.7545960545539856, -0.8671668767929077, -0.012146816588938236, -0.314974844455719, 0.4368763864040375, -0.4963034987449646, 0.9165886640548706, 0.5449766516685486, -0.2947310507297516, -0.026869023218750954, 0.333596408367157, 1.3213708400726318, 0.22892263531684875, -0.16239488124847412, -0.7388628125190735, 0.3889806866645813, 0.3921748995780945, 1.2600980997085571, -1.0613248348236084, 1.3303242921829224, -1.688863754272461, 0.30971741676330566], [-1.1315298080444336, -1.2215577363967896, -1.2732186317443848, -1.1051093339920044, -0.2678360939025879, -0.2808532118797302, 0.49880269169807434, 0.7343703508377075, -1.3580635786056519, -0.8462104797363281, 0.2599889039993286, -1.09890878200531, -0.034530967473983765, 0.0837421864271164, 0.3102380335330963, 2.389359951019287, 0.8658844828605652, 0.007049919571727514, 1.3018368482589722, 0.893126904964447, 0.9744853377342224, -1.7536991834640503, -1.5112555027008057, -1.9449294805526733, -1.9393924474716187, -0.18658308684825897, -0.8529738187789917, 1.399490475654602, -0.6224386096000671, 1.593950867652893, 0.4707065224647522, 0.4556131362915039, 1.085095763206482, -0.5002882480621338, 0.6215729713439941, 0.1581135243177414, -0.37065207958221436, -1.4604476690292358, 0.6516285538673401, 1.0018974542617798, 0.37279805541038513, 0.9198682904243469, 1.1255931854248047, -0.48908859491348267, 0.4247822165489197, -0.4393062889575958, -0.19056281447410583, -1.2978265285491943, -1.220699429512024, 0.3500536382198334, 1.8008896112442017, -0.30235135555267334, 0.5962023735046387, -0.36405906081199646, -0.6188050508499146, 0.5708637833595276, -0.49769023060798645, -0.24226613342761993, -0.2995574176311493, 0.13610225915908813, 0.7909199595451355, 0.21798193454742432, -0.4102500081062317, 0.9632558226585388, 0.6284383535385132, 1.2537569999694824, -0.4216001331806183, 1.0850303173065186, 0.038922056555747986, 1.3456203937530518, -0.3610166013240814, -1.4092328548431396, -0.18143697082996368, -0.10257736593484879, -1.042030692100525, 1.1684101819992065, 0.9198386669158936, 1.098202109336853, -0.23814234137535095, -1.3343734741210938, 1.4291112422943115, -0.39287006855010986, 0.46494221687316895, 0.01224072277545929, 0.6118326783180237, -0.3539482355117798, 2.0506489276885986, -0.9347587823867798, -1.9446412324905396, 0.8829346895217896, 0.9508246183395386, 0.4488998055458069, 1.6650121212005615, -1.7951111793518066, 0.912803053855896, -0.4752140939235687, 1.0937950611114502, -1.0583984851837158, -0.07374420017004013, -0.27163729071617126, -0.09826397895812988, -0.7623158693313599, 2.133103370666504, 0.11623212695121765, 0.19806653261184692, 0.07415284216403961, 0.8290308117866516, 2.365776300430298, 0.49146032333374023, 1.3189327716827393, 0.22141340374946594, 2.3849472999572754, -1.078192949295044, -0.3628138303756714, -0.11222352087497711, 0.36470693349838257, 0.5487961769104004, 1.0992454290390015, 0.008114888332784176, 2.401400566101074, 0.5344338417053223, -1.1723244190216064, -2.225773334503174, -0.7844481468200684, -0.24869178235530853, -0.42182403802871704, -1.0663461685180664, -2.154005289077759], [-0.029457496479153633, -1.414872169494629, -0.15867654979228973, -0.07014855742454529, 0.32859736680984497, 0.013409286737442017, -0.38907283544540405, -0.4761391580104828, 0.2402035892009735, 0.3247070908546448, -0.3780222535133362, -0.6020851731300354, 0.5880078673362732, -1.6638809442520142, -0.9164437651634216, -0.25794368982315063, 0.09914592653512955, 0.14668092131614685, -0.9828732013702393, -0.12000666558742523, -0.6385849118232727, -0.38374772667884827, -0.4475115239620209, 1.9827990531921387, 0.35085415840148926, 1.312662959098816, 0.8457545638084412, -0.004440664779394865, -1.0156581401824951, -1.6272531747817993, 1.0606900453567505, 0.4680362343788147, -0.05868348479270935, 0.1499023288488388, -1.7501404285430908, 0.33455848693847656, 0.2684917449951172, -0.9166257977485657, 1.6360641717910767, -0.30489927530288696, -0.6818612217903137, 0.2238350659608841, -0.7316038608551025, 0.4679708778858185, -2.26069974899292, 0.5032204389572144, -0.6475532054901123, -2.448660135269165, 0.9252369403839111, -1.9971321821212769, 0.6392831802368164, 0.10867702960968018, -0.711327850818634, -1.482113242149353, 0.25430065393447876, -0.1658930778503418, 0.8388254046440125, 1.804408311843872, 0.6709738373756409, 0.39567869901657104, 1.0292906761169434, 0.5077968239784241, -0.08203314244747162, 1.4876714944839478, -1.4059067964553833, 0.6386559009552002, -0.919012725353241, 0.2898748517036438, 0.24877715110778809, 0.023213759064674377, 1.4640758037567139, -0.4413793087005615, 1.0612612962722778, -1.900916576385498, -0.5524234175682068, 0.3099070191383362, 0.54441237449646, 1.0827865600585938, 0.9109528660774231, 2.018470525741577, 0.719594419002533, -0.3984307646751404, 0.1076289564371109, -0.16362784802913666, -0.18243102729320526, 1.5193579196929932, 1.6782747507095337, -0.10071813315153122, 2.0571048259735107, 0.9756865501403809, 0.5309285521507263, -0.050085075199604034, -0.14393427968025208, 1.089539647102356, -1.144593596458435, -0.4718865752220154, -0.8191481232643127, -0.3067112863063812, 2.324105739593506, -0.8636789917945862, -0.32417458295822144, 0.4531804621219635, -0.41455456614494324, 0.3110645115375519, 0.5848391652107239, 0.7539802193641663, 1.610719084739685, 1.480376124382019, 1.0610322952270508, -2.016252279281616, 0.35597923398017883, -1.1336796283721924, -0.7142131924629211, -0.993316113948822, -0.8485131859779358, 0.5618181824684143, -0.020183993503451347, 0.516292929649353, -1.26505708694458, -1.2363455295562744, -0.16458453238010406, -0.785377025604248, 0.08689778298139572, -1.3947534561157227, -0.5145244002342224, 0.08667608350515366, 1.186539649963379, -0.5139943957328796], [1.8635222911834717, 0.5498991012573242, -1.4855214357376099, -1.0084772109985352, 1.2780920267105103, 1.0118417739868164, -0.9424446821212769, 0.18649455904960632, 2.4975967407226562, 0.09641128033399582, -0.45249953866004944, -0.9415874481201172, -1.070381999015808, 1.0707411766052246, 0.8269683718681335, 0.17545713484287262, -0.18402735888957977, 0.2211751490831375, -2.268141508102417, -2.9527580738067627, -2.1322474479675293, -0.48835501074790955, 0.6247121095657349, 1.4316946268081665, -0.04024891555309296, 0.3653498888015747, 0.8153764605522156, -1.2064203023910522, -0.5489132404327393, -0.03344129025936127, 0.018750177696347237, -1.5196083784103394, 0.6844313144683838, -0.03621596097946167, 1.0516637563705444, -0.2787284851074219, -1.1148124933242798, -0.8986481428146362, 0.4544655382633209, 1.4710376262664795, -1.5076050758361816, 1.8623892068862915, 1.1222662925720215, -0.7962110638618469, -0.9386162161827087, -0.1570175439119339, 0.2653570771217346, -1.7095173597335815, 1.531137228012085, -0.8614873290061951, 1.2362371683120728, 1.0577958822250366, 0.1794072985649109, -0.5911920070648193, -1.2373383045196533, -0.5569965243339539, -0.19079287350177765, 1.0948610305786133, 0.41387930512428284, 0.056538451462984085, 0.36057591438293457, -0.09824575483798981, -0.02129487507045269, -1.7312476634979248, -1.2829045057296753, 0.6650561690330505, 0.5834290385246277, -0.6754665970802307, 0.8168312907218933, -1.6762202978134155, 0.6809033751487732, 0.73759925365448, 1.570361852645874, -0.22178351879119873, 0.09649024158716202, 0.7143650054931641, -0.11658559739589691, -0.9736907482147217, -0.10650845617055893, -1.7904489040374756, 0.2128356695175171, -0.8901856541633606, 0.29144948720932007, 1.1455415487289429, -0.19325900077819824, 1.5262999534606934, -1.3983100652694702, 1.1510180234909058, -0.18388248980045319, -0.15162092447280884, -0.12197603285312653, 0.02037326619029045, -0.6284776329994202, 1.2822775840759277, 0.13827969133853912, -0.021772101521492004, -0.5881026387214661, 1.759079933166504, -1.4407923221588135, 0.3865397870540619, -1.9052879810333252, 1.7633671760559082, 1.673324465751648, 0.7358869910240173, -0.06877369433641434, 0.03287040442228317, -1.0234100818634033, -0.03655502200126648, -1.2047955989837646, 1.285186767578125, 0.421752393245697, 0.8548610210418701, 0.5797402262687683, 0.7803260684013367, -0.7971253991127014, 0.2958759367465973, 0.6703137159347534, -0.006160592660307884, 0.6359078884124756, -0.3791073262691498, 0.5501931309700012, 0.5319703221321106, -0.4019240736961365, 0.37171056866645813, 2.0820627212524414, -0.5408459305763245, 0.12178362905979156, 0.4330950677394867], [0.9623730778694153, 0.8488894701004028, -0.20847804844379425, -0.4364154636859894, 0.7623011469841003, 0.27520933747291565, 0.0014358505140990019, -0.629276692867279, -0.39280450344085693, -0.4433525800704956, 1.5558900833129883, 1.1539417505264282, -1.9820597171783447, 0.8491525053977966, -1.3822747468948364, -1.429681658744812, 1.3836853504180908, 1.486114501953125, -1.257161021232605, -2.4233529567718506, 1.5956993103027344, -0.24681028723716736, -1.0759676694869995, -1.8184360265731812, -0.020867090672254562, 1.0950605869293213, -0.7556942105293274, 0.33502617478370667, 0.017593543976545334, -1.4805312156677246, -0.9817401766777039, -0.6639732718467712, 0.7054996490478516, 0.8430245518684387, 0.47947824001312256, 0.36924245953559875, -1.0099689960479736, -0.33163267374038696, 0.6848781108856201, 1.0751895904541016, -0.22537101805210114, 2.4396681785583496, 3.4119038581848145, 0.03114846535027027, -1.3174208402633667, -0.43785253167152405, 0.5067116022109985, -0.24142912030220032, -1.0949747562408447, 0.9424844980239868, 1.419789433479309, 1.460631012916565, 0.44872018694877625, 0.8372027277946472, 0.7950998544692993, -3.0929410457611084, 0.8164490461349487, -1.2125732898712158, 1.251942753791809, -0.1970561146736145, -0.7815788984298706, -0.5592177510261536, 1.1634275913238525, -0.37524813413619995, 1.830174207687378, 0.14158597588539124, -0.8736253380775452, -1.2968225479125977, 1.420419454574585, -1.0959725379943848, -0.6824131608009338, -0.12949249148368835, -0.45498788356781006, -0.23915350437164307, -1.562518835067749, -0.7261084914207458, -0.8012413382530212, -0.7504720687866211, -0.013656919822096825, 0.3688300549983978, -1.2505168914794922, 1.20072603225708, 0.5578180551528931, 0.24924428761005402, 1.0332653522491455, -1.143545389175415, 0.9613397717475891, -0.4674326181411743, 0.5614878535270691, -0.40729278326034546, 0.07457631081342697, 1.5621626377105713, -0.412948340177536, -0.5773804783821106, -1.0840116739273071, -0.45316046476364136, -0.9627107381820679, 0.37551945447921753, 0.5606573820114136, 0.4714665710926056, -0.42177385091781616, -0.5810952186584473, -0.9260896444320679, -0.01927102915942669, 2.125431537628174, 0.6469972729682922, 0.1742473989725113, -2.3779683113098145, 0.9133010506629944, -0.5671700835227966, -1.3121283054351807, -1.6762118339538574, -0.13929153978824615, 0.482930988073349, 1.2535960674285889, 0.052603159099817276, -0.7061067223548889, -1.0677281618118286, -0.0885361060500145, 1.2280387878417969, 0.7169011831283569, 0.18142856657505035, -0.35330966114997864, -0.44657936692237854, 0.105679951608181, 0.1791684329509735, -0.08078961074352264, -0.17606917023658752], [-0.8188232183456421, 1.9787116050720215, -0.09422978013753891, -0.4606245458126068, -0.8636672496795654, -0.9158102869987488, -1.3270647525787354, 1.090197205543518, 0.23804853856563568, 1.1311707496643066, -2.015401840209961, -1.5077872276306152, 0.7764330506324768, 0.5303319096565247, 0.5355815887451172, -0.8626717925071716, 0.5909442901611328, 0.5326366424560547, -1.4994890689849854, 0.8291932940483093, -1.0275501012802124, 1.4563746452331543, -0.965617835521698, -1.2475937604904175, -1.0363013744354248, -0.3493744730949402, 1.4397894144058228, 1.6387625932693481, -0.3269926607608795, 1.9109039306640625, -0.1750553399324417, -1.3678663969039917, -1.5130226612091064, -1.717705488204956, -1.0546300411224365, 1.6168320178985596, 0.8388783931732178, -0.19162890315055847, -0.1466141641139984, -1.0461128950119019, -0.4629126489162445, -0.7767021656036377, -0.8572418689727783, 1.849583387374878, -0.03563161939382553, -1.5618523359298706, -0.5914554595947266, 1.5506997108459473, -0.4604915678501129, -1.875043511390686, -0.3387037515640259, 0.1503899097442627, -0.29489487409591675, 0.538129448890686, 0.16828647255897522, 0.35127830505371094, 0.3605463206768036, 1.5082565546035767, 0.20722568035125732, 1.1120802164077759, 1.6155036687850952, -0.9913880228996277, 0.14624224603176117, 0.0065181502141058445, -1.0078152418136597, -0.7093846201896667, 1.5757684707641602, 0.11677480489015579, 0.05839161202311516, 1.129698395729065, 1.3280552625656128, -1.4388539791107178, -0.4714099168777466, 0.9456061124801636, -2.079953193664551, 0.4075712561607361, 0.8455086350440979, -1.0198729038238525, -1.0644400119781494, -0.5279465317726135, 1.6933811902999878, 0.010545467957854271, 0.789695143699646, 1.056532382965088, -1.4041965007781982, -0.018704552203416824, -0.7606413960456848, 0.7556580901145935, -1.0586681365966797, -0.5632861256599426, 2.821164131164551, 0.10250364243984222, -0.9539315700531006, 1.396005392074585, -0.9410228133201599, 0.7067270874977112, 0.27987465262413025, 1.143181562423706, -0.4301862120628357, 0.30776268243789673, -0.5616717338562012, 1.3312991857528687, 0.4619859755039215, 0.3774961233139038, 1.6502865552902222, -0.0567195788025856, 0.8530719876289368, -1.5482500791549683, -0.6203118562698364, 0.42599084973335266, -0.07222530245780945, 0.2571491003036499, -0.22555963695049286, -0.17848651111125946, -0.9841052293777466, 0.6197176575660706, -1.8321388959884644, -0.3724214732646942, 0.37527987360954285, 0.22258158028125763, -1.0697249174118042, 0.8180323839187622, -1.196822166442871, 0.49807941913604736, 0.8843483328819275, 0.15413005650043488, 0.6530773639678955, -0.44128844141960144], [-0.31801897287368774, -1.1036125421524048, -0.008039710111916065, 0.4871252179145813, 0.16911263763904572, -0.6900609135627747, -1.5398586988449097, -0.6389157772064209, 0.07995574176311493, 1.7008739709854126, 0.11359170079231262, 1.1600443124771118, -0.7336342334747314, -1.701720952987671, -0.22441674768924713, 0.48428118228912354, -1.1113910675048828, -1.3000564575195312, -0.9368041753768921, -0.9911453723907471, -0.072026327252388, -0.5912903547286987, -0.230865478515625, 0.006141458172351122, -1.9232298135757446, -2.011746644973755, 1.306422472000122, -0.3461686372756958, 1.5930334329605103, 1.4881341457366943, -0.6408027410507202, 0.8951502442359924, -0.06070427596569061, -0.3250686824321747, 1.0926052331924438, 2.084240198135376, -0.3136957883834839, -2.1460025310516357, -1.3568307161331177, -1.8621282577514648, 0.7359263896942139, -1.5396336317062378, -0.551045835018158, -1.6235370635986328, -2.1307501792907715, -0.6555992364883423, 0.8474393486976624, 0.03790992498397827, -2.101142406463623, -1.0010722875595093, 0.7048099637031555, 0.01959092542529106, 0.2197246104478836, -0.2630089521408081, -0.4736562967300415, -0.630561113357544, -0.8432883024215698, 0.29562482237815857, -1.7262948751449585, -0.8922199010848999, -0.8431394100189209, 1.5112061500549316, 0.3714594542980194, -0.7084863781929016, -0.6240357160568237, 0.6588244438171387, -0.40133196115493774, 0.8884839415550232, 0.15948042273521423, 1.0478399991989136, -0.09659262746572495, -0.9297373294830322, -0.13262030482292175, -1.3349111080169678, 1.652390480041504, -0.1595809906721115, 0.2933404743671417, 1.3892539739608765, -0.24489019811153412, 0.5230438709259033, 0.9038320183753967, 1.6996501684188843, 0.5281548500061035, -0.14230014383792877, 0.16291216015815735, 1.4831643104553223, -0.9317942261695862, -0.8603411316871643, -2.4217476844787598, -0.4417811334133148, -1.5756224393844604, 0.5050252676010132, 0.8994033932685852, -1.4108120203018188, -1.3993635177612305, -0.5784168839454651, 0.022403929382562637, 1.5081267356872559, 0.5470779538154602, 1.2004640102386475, -1.935034155845642, -0.5582593083381653, -0.08539003133773804, -1.0855094194412231, -1.2102218866348267, 0.1400539129972458, 0.7929991483688354, -1.639716625213623, -0.9723058342933655, -1.0127884149551392, -0.13226550817489624, -1.7690398693084717, 1.1267825365066528, -0.0437639020383358, -0.24800734221935272, -0.28809696435928345, 0.3466580808162689, 3.272193431854248, 0.09291601926088333, -0.7209922671318054, 0.9188255667686462, 1.6554951667785645, 2.0033113956451416, -0.644845724105835, -1.0967185497283936, 0.6193760633468628, -0.35302993655204773, 0.7737919688224792], [-0.9235920310020447, -0.9434120655059814, -1.392369270324707, 1.121032953262329, -0.17533767223358154, 1.160327434539795, -1.2604784965515137, 0.12315990030765533, -0.6787019371986389, -0.9216710925102234, 1.4177254438400269, -0.3438977003097534, 0.5636471509933472, -0.04842868819832802, 1.3256069421768188, 0.6962624192237854, -0.5798783898353577, 1.1160516738891602, -1.2779191732406616, 1.4076995849609375, -0.3668489158153534, 1.9047009944915771, 0.4347226023674011, 0.31679093837738037, -0.7450881004333496, 1.661648154258728, 0.24813687801361084, 1.063033103942871, 0.616283118724823, -0.6840466260910034, 0.6052418351173401, 1.364430546760559, 1.02117121219635, -0.3638099730014801, 0.6156219244003296, -1.9313840866088867, 1.4484177827835083, -0.42673155665397644, 0.30991414189338684, 0.3964586853981018, -0.12637637555599213, -1.2859896421432495, 0.44660061597824097, 1.1982039213180542, 0.903770923614502, -0.03786173462867737, 1.1301579475402832, 0.41285189986228943, 0.7046777606010437, -0.6244146227836609, 0.632455587387085, 0.15457943081855774, 2.6694247722625732, -1.234229326248169, -1.073921799659729, -0.7912549376487732, -0.9267538189888, -0.37097856402397156, 0.23852872848510742, -0.5851861834526062, -0.006190600339323282, -1.0034615993499756, 0.555461049079895, -0.9364922046661377, 1.0433716773986816, -1.1599016189575195, 0.39812955260276794, 0.07313097268342972, -1.97102952003479, 0.4731876254081726, -0.1701221466064453, -1.6154855489730835, 0.6171606779098511, -0.3628816604614258, -0.8912054300308228, -0.4364698529243469, -0.3474838137626648, -1.7388705015182495, 0.03966455161571503, 0.15233354270458221, -0.20705987513065338, -0.5423173308372498, 0.8259856104850769, -0.06605633348226547, 0.15932585299015045, -0.44501057267189026, -0.7993577718734741, 0.6343849301338196, 1.206291913986206, 1.06146240234375, -1.3963004350662231, 0.7882596254348755, 0.18028990924358368, -2.241177558898926, 0.09166588634252548, -0.9228873252868652, -1.6655614376068115, 0.37338075041770935, 2.179100275039673, -0.5909691452980042, -0.7502944469451904, 0.324327677488327, -0.05238179489970207, -0.4858740270137787, -1.466744065284729, 0.4269223213195801, -0.6924116015434265, 0.5843307971954346, -0.3540024757385254, -0.7405219674110413, -0.045486729592084885, -1.2834978103637695, 2.6355159282684326, -0.07793285697698593, -0.21278643608093262, 0.5621623396873474, 0.1264796406030655, -0.7645182609558105, -0.3633672893047333, 0.8312807679176331, 1.2137856483459473, -2.317164182662964, -0.5326932072639465, 1.6668912172317505, 0.6914647817611694, 1.45122230052948, 0.6735293865203857, 0.5784791707992554], [-0.3087235987186432, 1.2899099588394165, 1.308087944984436, 2.4236252307891846, -1.0030094385147095, -0.521787703037262, -0.32079142332077026, 1.216290831565857, 0.3688278794288635, 0.9563932418823242, 0.544840931892395, -1.1230716705322266, 0.2518806457519531, -1.2367115020751953, -0.44508856534957886, 0.6166369318962097, -0.046321846544742584, -0.5914353728294373, -0.5666512846946716, -0.1433982402086258, 0.014075271785259247, -1.5929516553878784, 0.5079955458641052, -1.182711124420166, -0.6151642799377441, -0.5255050659179688, 0.6378269791603088, 0.36145877838134766, 0.8846986293792725, -0.6495583057403564, 1.0804152488708496, 1.134125828742981, -1.1944164037704468, 0.2882539629936218, 0.05972208455204964, -2.727656126022339, 0.5210485458374023, 1.075669765472412, -0.3386518955230713, 0.20255008339881897, 0.34619319438934326, -0.9226126670837402, -0.4730491042137146, -0.36094048619270325, 0.06767819821834564, 0.20998995006084442, 0.7614790201187134, -0.6422020792961121, 1.3473390340805054, 0.41755348443984985, -0.06436853855848312, -0.49093538522720337, 0.7105376720428467, 0.9599736332893372, 0.6915260553359985, 1.9068697690963745, 0.1417267769575119, -0.030950386077165604, -0.27815884351730347, -0.8837284445762634, 0.29086601734161377, -0.5086202025413513, 0.1056889146566391, 0.9163206219673157, -0.13226142525672913, 0.5770536661148071, -1.5772055387496948, -0.10839953273534775, -0.7594778537750244, -1.439893364906311, 0.1987556666135788, 0.4748108983039856, 0.9745783805847168, -0.10561306029558182, 0.24695627391338348, 0.6499912142753601, -2.086442232131958, 1.2264851331710815, 2.1056737899780273, 0.7154433131217957, -1.2974530458450317, -0.02015487477183342, 1.4790043830871582, -0.5820042490959167, -0.3859499990940094, 0.6303905248641968, -1.7296775579452515, -0.273853063583374, 1.0955160856246948, -1.8609706163406372, -1.7444310188293457, -0.602727472782135, 1.6495269536972046, -0.17067451775074005, 1.528654932975769, 1.333723783493042, -0.06444180756807327, -1.2108960151672363, 0.23500359058380127, 1.5028330087661743, 1.4673856496810913, 0.03269689902663231, -0.0932120755314827, 0.6792684197425842, -0.6841886043548584, 1.233616828918457, -0.3281595706939697, 1.2405844926834106, -1.1725287437438965, -0.546207845211029, -0.4449257552623749, -1.8537406921386719, 0.7488391399383545, -0.5216570496559143, -0.43284910917282104, -0.26264506578445435, -1.6199015378952026, 0.04018792137503624, 1.9319415092468262, -0.8505977392196655, 1.7771015167236328, -0.29231300950050354, -0.428227961063385, 1.9518989324569702, 0.5775654315948486, -0.9586576819419861, 0.40405982732772827, 0.35480669140815735], [0.3243415951728821, 0.015437308698892593, -0.09570472687482834, -2.202720880508423, 0.5065011382102966, 4.167205810546875, 0.3222767412662506, 0.3097340166568756, 2.0728600025177, 0.1458677053451538, 0.7050414085388184, 0.9325432777404785, 0.4285818338394165, -1.2576614618301392, -0.38991615176200867, 1.2945398092269897, -0.7576549053192139, -1.6155023574829102, 0.4867538809776306, 0.7687456011772156, 1.5648561716079712, -1.0581614971160889, 0.11449185013771057, -1.6712491512298584, -1.3647114038467407, -0.123384490609169, 0.1966504156589508, 0.14156220853328705, -0.7830835580825806, 0.950206995010376, -0.8421997427940369, -0.31880080699920654, -0.1700449436903, -1.2859433889389038, -0.734442949295044, -0.11285139620304108, 0.6634682416915894, 0.6672040820121765, -0.21611660718917847, 0.25735917687416077, -0.09384065866470337, 0.4687565863132477, 2.814225673675537, 0.2253987342119217, -1.962662696838379, -2.4763550758361816, -0.9848080277442932, 0.46076375246047974, -1.2646093368530273, -0.8562649488449097, 0.15180528163909912, 1.2735260725021362, 0.2575734853744507, -0.4386405050754547, -0.22387614846229553, 0.7434605956077576, -0.20863784849643707, -0.72197026014328, 1.9498119354248047, 0.36992064118385315, -1.6504071950912476, 0.45400136709213257, -1.0539921522140503, -1.4754358530044556, 0.11133425682783127, -0.8898624181747437, 0.42923253774642944, 0.9539706707000732, -0.7833917140960693, -0.08100719749927521, -1.7316176891326904, -1.0578261613845825, 0.5442155599594116, 0.4952246844768524, 0.8750633001327515, -0.784694254398346, -0.6582067608833313, -0.3164607882499695, 2.613175392150879, 0.17624260485172272, -0.7669767737388611, -2.5959842205047607, 0.6415932774543762, 0.18578225374221802, 1.032446265220642, 1.1683748960494995, -1.7117847204208374, 0.9070557951927185, -0.4800270199775696, -0.2874332070350647, 0.5064648985862732, -0.22247987985610962, -1.3771430253982544, -0.5197968482971191, -0.525039553642273, 1.0136723518371582, -0.5220993757247925, 0.1264733076095581, -1.1013708114624023, -1.0318732261657715, -0.9334129691123962, 1.3450418710708618, -0.5192223191261292, -0.1742355227470398, -1.1003037691116333, 0.694823682308197, 0.42464590072631836, -1.0533246994018555, 0.9013418555259705, -1.7840677499771118, 0.6588186025619507, 0.07556045055389404, 0.6884803771972656, -2.1083824634552, -0.6426331996917725, -0.07480666786432266, 0.1647985428571701, -0.777332603931427, 0.9196300506591797, -1.271220088005066, 0.19921785593032837, -0.43695876002311707, 1.4977692365646362, -1.6666020154953003, 0.34179699420928955, 0.33089831471443176, 0.47524023056030273, -0.6637739539146423], [-0.4087359607219696, 0.5858264565467834, 1.8779109716415405, 1.4064258337020874, 0.8744725584983826, -0.8234686851501465, 0.9641016721725464, -1.772456407546997, 1.069523572921753, 0.33772924542427063, 0.2791405916213989, -0.1035822182893753, 1.3801378011703491, -0.3790919780731201, 0.06494453549385071, -0.34487542510032654, -1.1835650205612183, 1.2069586515426636, 1.4880539178848267, 0.5369648337364197, 1.4493578672409058, 0.037560220807790756, -0.5091078877449036, 2.1363210678100586, -0.47469040751457214, -1.0743041038513184, -1.6888624429702759, -0.3771705627441406, -1.1688348054885864, 0.21064263582229614, 0.8502121567726135, 0.29488006234169006, -1.7399625778198242, -0.3764920234680176, -0.7747926712036133, 0.17082200944423676, -0.35309576988220215, 1.1812283992767334, 1.2910736799240112, 0.7329305410385132, -1.057878851890564, -0.1768231838941574, -2.3427584171295166, 0.04857964068651199, -0.7992382645606995, -0.5570293068885803, -0.6898593902587891, -3.0253612995147705, -0.3637806177139282, -0.4489816725254059, 1.4348936080932617, 0.33219999074935913, -0.7882685661315918, 0.46741983294487, -0.31012779474258423, -1.0972509384155273, -0.557235836982727, 0.968801736831665, 0.05382164567708969, -1.0107626914978027, 0.3026731312274933, -0.23979628086090088, 0.4751669764518738, -1.0192539691925049, 1.0590109825134277, -0.11826649308204651, -1.1585652828216553, 0.39137667417526245, -0.111324243247509, -1.4501290321350098, 0.06549480557441711, 0.5050642490386963, 1.6219700574874878, 0.4931613802909851, 1.1999744176864624, -0.2895985543727875, 0.7743031978607178, 1.5873774290084839, -0.2309631109237671, -0.12484928220510483, -0.4270157814025879, 0.25034207105636597, 2.307560920715332, 1.5554507970809937, -0.06940901279449463, 1.7211040258407593, -1.4119458198547363, 1.6008775234222412, 0.11161316931247711, -0.5324811339378357, -0.8956053853034973, 0.12042263895273209, -0.7703935503959656, -0.7995144724845886, -0.6607893109321594, -0.2617887258529663, 0.8665224313735962, 0.798740029335022, 0.6709139347076416, -0.5474250912666321, -0.8626905679702759, 0.12143433839082718, -0.08427655696868896, 0.06174388900399208, -0.48961341381073, 0.16436946392059326, 0.585639476776123, -1.221198558807373, -2.463341236114502, 0.9100375175476074, 2.666260242462158, 1.2019038200378418, 0.9876737594604492, -0.5046825408935547, -0.010638384148478508, 0.988588809967041, -0.326276957988739, 0.3741534352302551, 2.1595065593719482, -0.3826020061969757, -0.48789042234420776, -0.7108191251754761, -0.7871112823486328, 0.6233690977096558, -1.088672161102295, -0.2648937404155731, 0.24825215339660645, -0.009703896939754486], [0.6559037566184998, -0.4268377125263214, 0.39305049180984497, -0.8625838756561279, 0.5453738570213318, 0.4008963406085968, -1.4819550514221191, -0.7084718346595764, -0.030900128185749054, 0.5364237427711487, 1.8340132236480713, 2.4345836639404297, 1.2938815355300903, 0.6063631176948547, 0.4748383164405823, -2.448216438293457, -0.5852175354957581, -0.7939383387565613, 1.234555721282959, -2.050693988800049, -0.656848132610321, 0.17230813205242157, -0.6876431703567505, -1.3405095338821411, 1.235229730606079, -0.27915358543395996, -0.6301853656768799, -0.48193520307540894, -1.8114526271820068, -0.6391791105270386, -0.6212707757949829, 1.4562431573867798, -0.8377465009689331, 0.37277671694755554, -1.1574833393096924, 0.052313193678855896, 1.2824108600616455, 0.6538106203079224, 1.8491778373718262, -0.03939221054315567, -0.2972443401813507, -0.9665132164955139, 0.1359180212020874, -1.0376261472702026, 0.28166863322257996, -0.1233510673046112, 0.6029323935508728, -2.6871728897094727, 2.065639019012451, 0.18536287546157837, -1.6511709690093994, -1.9261221885681152, 0.5689292550086975, 0.5313403606414795, 0.8625783920288086, 0.5859656929969788, 0.11954455077648163, -0.5622360706329346, 0.14438898861408234, -0.020707685500383377, -1.5768756866455078, 0.4191622734069824, -1.1989678144454956, 0.7022201418876648, -0.4113747179508209, -0.49929970502853394, 0.9830791354179382, -1.130050778388977, 1.736675500869751, 0.5607826709747314, 1.6317263841629028, 0.7882132530212402, 0.21928510069847107, 0.9553471207618713, 0.37344956398010254, -0.13827668130397797, 0.22643795609474182, 1.681378960609436, 0.49702566862106323, 0.28783532977104187, 1.0210003852844238, 1.770043969154358, -0.6858842372894287, 0.13423581421375275, 0.1168954148888588, 0.2335282415151596, 0.5280119776725769, -0.46859651803970337, -1.165941834449768, 0.4860724210739136, 1.246571660041809, -1.1976432800292969, -0.8238363265991211, 0.8685659766197205, 0.09715834259986877, -1.2526084184646606, -1.7865992784500122, 0.0023447589483112097, 0.47137877345085144, -1.6580195426940918, -1.0469051599502563, -0.0957634225487709, -0.18063059449195862, -1.0565992593765259, 0.4192942976951599, -2.170121908187866, 1.393223524093628, 0.3047308921813965, 1.0002373456954956, 1.4875683784484863, -1.4738492965698242, 0.059130363166332245, -0.9481194019317627, -1.5545597076416016, 0.28834399580955505, -0.09555614739656448, 0.3941015303134918, -0.13470692932605743, -0.4410698413848877, 1.9157766103744507, -0.4412045478820801, 1.5111234188079834, -2.0468783378601074, 0.015590139664709568, -0.04037958011031151, 0.2941659986972809, 0.0620553083717823, -1.012992262840271], [-0.0031451599206775427, 0.08223199844360352, 1.0358734130859375, -0.26109927892684937, -0.3401108980178833, -0.3908464014530182, 1.889465570449829, -1.4131778478622437, 1.4549005031585693, 0.7840468883514404, 1.4375029802322388, -0.5461190342903137, 0.804932177066803, -0.9714084267616272, 1.0486663579940796, 1.155489206314087, -0.22631461918354034, -3.175194263458252, -1.2217060327529907, 1.9578797817230225, 0.03846035152673721, 1.4918663501739502, -0.594150960445404, 0.029251649975776672, 1.1272250413894653, 1.6284757852554321, -0.7863754034042358, 0.8241734504699707, -1.0247480869293213, -0.5644903182983398, 0.36578479409217834, 2.1742708683013916, -0.5198559165000916, -1.1536122560501099, 1.785590648651123, -0.5644139647483826, 0.9460642337799072, 1.2229297161102295, 0.37328726053237915, 0.23345696926116943, -0.22234120965003967, -0.46969661116600037, -0.6405660510063171, -1.2932206392288208, 0.3720291256904602, -0.7166185975074768, 0.9042131900787354, 1.225283145904541, 1.8695809841156006, -0.014692376367747784, 0.09036017954349518, 0.5282673239707947, -0.8997334837913513, 1.4735702276229858, 0.01918351650238037, 0.34316515922546387, -1.0030910968780518, -0.0647997185587883, -1.0761350393295288, 1.2449898719787598, -1.1428396701812744, -1.1960136890411377, 0.3401263952255249, -0.4346303939819336, -0.9363052248954773, 0.7466704845428467, -0.39519166946411133, -1.1605443954467773, -1.336423397064209, -1.9113984107971191, 0.8727854490280151, 1.3384324312210083, -0.3648153841495514, 0.7014843821525574, -0.6900734305381775, 0.2204200178384781, -0.7566168308258057, 0.6860210299491882, -0.026880918070673943, -0.8346561789512634, -0.4777967929840088, -2.605442523956299, 1.9266588687896729, 0.19857698678970337, -1.4584619998931885, 0.11449350416660309, 0.887732207775116, 0.989472508430481, -0.49323952198028564, 1.0485821962356567, -1.5464049577713013, -0.9865034818649292, 0.020760977640748024, -1.938530683517456, -0.11782736331224442, 0.022173168137669563, 0.5147164463996887, 0.5233492851257324, 0.8062058091163635, -1.9478968381881714, 1.142939805984497, 0.9331157803535461, 0.2479175627231598, -0.9231756925582886, -0.3650491237640381, -0.2880319356918335, 1.291776180267334, 0.02876215986907482, 0.21359363198280334, -0.4895077645778656, 0.3304283022880554, -0.04588361829519272, -1.2884094715118408, 0.5453521609306335, -0.12638990581035614, 0.5608648657798767, -0.4771922826766968, 0.37385353446006775, 0.6241872906684875, 0.880945086479187, -0.13700281083583832, -0.6712579727172852, -0.9152384400367737, 0.6955183744430542, -0.124404676258564, -0.0768345296382904, -0.5550076365470886, -2.0861783027648926], [0.7958112359046936, -0.7997747659683228, 0.7359886169433594, -0.2513444423675537, -1.524164080619812, -0.5564390420913696, 0.3394278287887573, -2.420043706893921, 1.990836262702942, 1.6161019802093506, 0.2693740725517273, -0.251983106136322, 1.1690953969955444, 0.688943088054657, -0.5986066460609436, 0.7873536348342896, 0.2899291217327118, 0.4678974449634552, -0.39445826411247253, 2.142622947692871, 0.7684230208396912, -0.3965446352958679, -0.7678486704826355, 2.2940495014190674, 0.4610915184020996, -1.2723653316497803, -1.1102957725524902, -0.3469470739364624, -1.2604140043258667, -1.8566640615463257, -2.9250192642211914, 1.030051827430725, -0.7901737093925476, -0.3814235031604767, 0.19703823328018188, 0.30504587292671204, 0.27453941106796265, 0.5547336935997009, 1.2490594387054443, -0.1438155472278595, 0.42675232887268066, -2.484389305114746, 1.0028698444366455, 0.6461851000785828, 2.0532138347625732, -0.06951991468667984, -0.5667405724525452, -0.059755291789770126, -0.3960285186767578, 0.44668757915496826, 1.7170631885528564, -0.5107032060623169, 0.23416534066200256, -1.4317902326583862, 0.9354521632194519, -1.9900929927825928, 0.8172962069511414, -0.792061984539032, 0.07294544577598572, -2.005326986312866, -0.3178275525569916, -0.636457085609436, 0.4027453362941742, -1.8290364742279053, -2.0544514656066895, -0.37015119194984436, -0.9918738007545471, 3.124009132385254, -0.34467193484306335, -0.015339825302362442, 0.4970197379589081, 0.6497238874435425, -0.2631169259548187, 0.32258471846580505, -0.0486484169960022, -0.8129324913024902, -0.9284681081771851, -1.9989542961120605, 0.20731092989444733, -0.6877231597900391, 0.6674706935882568, -1.3634834289550781, -0.25492018461227417, 2.060357093811035, 0.014756161719560623, -1.80897855758667, -0.30662766098976135, 0.4662407338619232, 1.936745524406433, -0.9394984245300293, 1.0868687629699707, 0.33918482065200806, 0.6005898118019104, 0.7121475338935852, 1.5212589502334595, 1.312849760055542, 1.0315343141555786, -1.0063536167144775, -0.9110303521156311, 0.011521674692630768, 0.34875622391700745, -0.8466292023658752, 1.1178561449050903, 0.6952405571937561, -0.2764241695404053, -1.5785695314407349, 1.1827231645584106, -0.8602499961853027, 0.5387913584709167, 1.6116453409194946, 0.022895997390151024, -0.4192247986793518, -0.840294361114502, -1.4386987686157227, 0.39891451597213745, 2.3232827186584473, -0.4560023248195648, 0.48129773139953613, -1.0261353254318237, 2.025765895843506, 0.948205292224884, 0.02447197027504444, 0.1323169767856598, 0.45989230275154114, 1.2040963172912598, 1.054901123046875, 0.5936575531959534, 0.36125245690345764]], [[-1.2589435577392578, -0.19699649512767792, -0.398750901222229, -0.9587357044219971, -0.1271073967218399, -0.7481719255447388, 1.2984238862991333, 0.11412165313959122, -0.1567203849554062, -2.1687071323394775, 0.44232887029647827, -1.3100528717041016, -0.8188169002532959, -0.8261989951133728, -0.11409012973308563, 0.18933774530887604, -1.6589686870574951, 0.3808494210243225, 0.35054245591163635, 0.8028475046157837, -0.20031975209712982, 0.17589472234249115, -0.0023361460771411657, 0.37273433804512024, 1.0449035167694092, -1.516155481338501, -1.7917544841766357, -0.7676394581794739, 0.6822829842567444, 0.2370678037405014, -0.7805464267730713, -0.5445437431335449, -0.7170165777206421, 0.2960444390773773, 0.07527466118335724, -0.25847330689430237, -1.2332885265350342, -1.4950494766235352, 0.6928369998931885, -0.1113823652267456, 0.8621842861175537, -0.3484310209751129, -0.37328970432281494, -0.8541504740715027, 0.8504679203033447, -0.233438178896904, -0.008009371347725391, 0.22193096578121185, 0.0928807407617569, 1.2537431716918945, 1.0366606712341309, -0.7582732439041138, 0.1786165088415146, -0.8872256875038147, -0.42714476585388184, -1.4722250699996948, -0.21796824038028717, -0.3692649006843567, -1.0641051530838013, -1.8205431699752808, -0.4839494824409485, -0.2693183422088623, 1.819100022315979, 0.20571669936180115, 1.0568407773971558, -0.2586432993412018, 0.8315978646278381, 1.1964848041534424, -1.1192764043807983, 0.48539790511131287, 0.9453612565994263, 0.0680459514260292, 1.2251191139221191, 0.7247738838195801, -1.171426773071289, 0.5345793962478638, 0.5315229296684265, 0.003938966430723667, -1.107108235359192, -1.6518217325210571, 0.9989053606987, 2.3057820796966553, -0.4227147102355957, 0.6009110808372498, -1.1244733333587646, -0.7592560648918152, 1.4886847734451294, -0.48480772972106934, 0.024746114388108253, 0.4808177053928375, 1.3718968629837036, 0.6260755658149719, 0.8043240904808044, 0.9655159711837769, 0.5655470490455627, -0.038383759558200836, -0.23035882413387299, 0.3845987915992737, 0.5344808101654053, -0.020752709358930588, -0.29085466265678406, -0.14261619746685028, -1.6740115880966187, -0.16523095965385437, 0.39194366335868835, -1.4939558506011963, -0.11959809064865112, 0.6358680725097656, 0.3564881980419159, 0.3232024013996124, -0.6789329648017883, -0.7735521793365479, -1.0842152833938599, -0.6086491942405701, -0.09991699457168579, -0.7055332660675049, -0.7349187731742859, -0.7766923308372498, 0.47549140453338623, -2.1749017238616943, 1.0000123977661133, 0.700461745262146, 0.1521635800600052, 1.140546202659607, -0.6850703358650208, -0.7992282509803772, 0.09156778454780579, 0.8731613755226135], [1.2339413166046143, -1.4654085636138916, 1.1887322664260864, -1.0095607042312622, -0.3379390239715576, 0.9273539185523987, 0.566438615322113, 1.0878198146820068, 0.36965861916542053, 0.0939774289727211, 0.6979000568389893, -1.1226236820220947, -0.5379952192306519, 0.3904878497123718, 1.3194451332092285, -0.6050043106079102, -0.7194542288780212, 0.12762607634067535, 1.604215383529663, 1.5995503664016724, 0.4599791169166565, -1.1899977922439575, -0.019257664680480957, 0.6868289113044739, -0.5029391646385193, -0.7104091048240662, -1.0018935203552246, -1.8473793268203735, -0.9920760989189148, 0.4900748133659363, -0.2840851843357086, 0.07092025876045227, -0.22861024737358093, 0.33353087306022644, -0.37598782777786255, -0.6783225536346436, 0.07791285961866379, -0.5655706524848938, -0.3689725995063782, 0.22498150169849396, -1.3233352899551392, 1.0726715326309204, -0.31446343660354614, 0.4761717617511749, 0.9447047710418701, 0.45266881585121155, 0.7295567393302917, -0.6909409165382385, 0.7973021268844604, -0.6968880891799927, 1.3417458534240723, -0.29325130581855774, -0.07884206622838974, -0.37681540846824646, 0.19886581599712372, 0.5296636819839478, -0.7023754119873047, -0.43062371015548706, -0.01437170896679163, -0.6289149522781372, 1.059820532798767, 0.8221667408943176, 0.8424498438835144, -0.16770750284194946, -0.37233400344848633, -1.0413509607315063, 0.7492767572402954, 0.46709999442100525, -0.3028395473957062, -0.7673400640487671, 1.1127078533172607, -0.9732344150543213, 1.4186972379684448, 0.6226288080215454, 0.43010878562927246, -0.5424754023551941, 0.8145977258682251, 0.05943011865019798, -0.32745087146759033, -0.909474790096283, -0.043509915471076965, -0.6981567144393921, -0.6725155711174011, 0.295147180557251, 1.399468183517456, -1.555018663406372, -2.132687568664551, 1.3236075639724731, 1.2001800537109375, -0.3086149990558624, 0.5916662812232971, 0.11813399940729141, -0.0037685621064156294, 0.045769836753606796, 1.5439510345458984, -0.03888928145170212, -0.8977900147438049, 0.50658118724823, 0.7164574861526489, -0.56959068775177, 1.4945836067199707, -0.2120957225561142, 1.1782792806625366, 0.7703239321708679, 0.07211333513259888, 1.1640278100967407, 0.2766512632369995, 0.8885301947593689, 0.30614539980888367, -0.8803881406784058, -1.257310390472412, 0.11984928697347641, -0.08755193650722504, -0.38944244384765625, -1.4617382287979126, 0.14533449709415436, -0.13385902345180511, 0.8993995785713196, 0.2775101065635681, -1.0047701597213745, -1.104314923286438, 0.1057816818356514, 1.7351139783859253, 0.11978071928024292, 0.9786056280136108, 0.6658205986022949, 0.0696958377957344, 1.0251374244689941], [-0.31628021597862244, 0.08931713551282883, -0.4974367916584015, -1.8747179508209229, 0.4729136526584625, -1.7591785192489624, -0.11747387051582336, 0.4592646658420563, 0.0486791618168354, 1.3835852146148682, 1.664156436920166, -0.25210487842559814, -0.7145796418190002, 1.393014907836914, -0.9832071661949158, -1.4435943365097046, 0.775087296962738, 1.4212391376495361, -0.7526445388793945, 1.2612111568450928, 1.171151876449585, 0.9296427369117737, -0.9503833651542664, -0.5069589614868164, -0.31064873933792114, -0.7671958804130554, -0.8626927137374878, 0.2596735656261444, 0.4459022283554077, 0.25537341833114624, -2.17045259475708, 0.633768618106842, -0.41923007369041443, -1.4358826875686646, 0.2754841446876526, -1.09380304813385, -1.0480989217758179, 1.6787633895874023, 0.8066596984863281, -0.5623830556869507, 0.42689836025238037, 0.7901906967163086, -0.164365753531456, 1.3394941091537476, -0.626461923122406, -1.1201311349868774, 0.3264836370944977, 0.5057755708694458, 0.2160889208316803, 0.38912343978881836, -0.9344250559806824, 0.06591479480266571, -1.402959942817688, -0.033459629863500595, -1.051672101020813, 1.36225163936615, 0.20318004488945007, -0.9206568598747253, -0.1548401564359665, 0.29525625705718994, 0.8045347929000854, -0.47328999638557434, -0.01229005679488182, 1.2116223573684692, -1.0964233875274658, 0.4141461253166199, 0.7612835764884949, 0.8390480875968933, -0.1210474893450737, -0.7527174949645996, 0.8345014452934265, 0.4123820960521698, -0.4702219069004059, 1.099543809890747, -0.8743793368339539, 0.23251040279865265, -1.592147946357727, -0.8578248620033264, -0.727105438709259, 0.4377683997154236, 1.7510966062545776, 0.6892760992050171, 0.29286524653434753, 2.007040500640869, 1.201265573501587, -0.5953064560890198, 0.40869152545928955, 0.6605613231658936, 0.08632925897836685, 0.2977343499660492, -0.6304131150245667, -0.7485027313232422, -0.3547171950340271, 1.3353323936462402, 1.0522559881210327, 0.3596850037574768, 0.38074252009391785, -1.7069497108459473, -1.5944737195968628, -1.682751178741455, -1.6174763441085815, 0.6946303844451904, -0.18046188354492188, -0.9985820651054382, 1.2351281642913818, 0.4459129869937897, 1.822957992553711, -2.0570263862609863, -0.7100716829299927, -1.743041753768921, -1.4699679613113403, 0.6577796936035156, -1.0857735872268677, 1.2737171649932861, 0.7915096879005432, 0.4336259365081787, 0.06704524904489517, 1.544189691543579, 0.32907968759536743, 0.41575804352760315, 1.003401517868042, 1.2113256454467773, 1.316055417060852, -0.7369652986526489, -0.6967781782150269, -0.493590384721756, 0.219572514295578, -0.04556716978549957], [2.134887933731079, 1.7481762170791626, -1.157083511352539, 0.5227275490760803, 0.23331062495708466, -0.7004566788673401, 0.4400951862335205, 1.1103084087371826, -0.21283291280269623, -0.04650122672319412, -1.1327776908874512, -1.196610927581787, -0.8551636338233948, 0.099252849817276, 0.19544635713100433, -1.357153296470642, 0.9935499429702759, 2.9445321559906006, -0.26113423705101013, -0.6280916333198547, 0.4309006333351135, -1.930724859237671, 0.05048755556344986, -0.3916396498680115, -1.5619832277297974, 0.1556060016155243, 2.1065988540649414, 0.6012328267097473, -1.2583215236663818, -0.31419453024864197, 1.043544888496399, -0.3851580023765564, -1.0424702167510986, -0.32824525237083435, -0.19137296080589294, -1.5910134315490723, -0.03083980269730091, -1.9415006637573242, -0.05084050074219704, -1.0229612588882446, 0.10241146385669708, -0.05402239039540291, -0.6366304159164429, 0.8142478466033936, 0.4682080149650574, -2.6010730266571045, -0.24485638737678528, 0.02436547912657261, 0.03055541217327118, 0.43272078037261963, -1.3347153663635254, -1.0763769149780273, 0.5988278388977051, 0.5772961378097534, 0.3463962972164154, 1.5724694728851318, -0.9241601228713989, -0.04786065220832825, 0.663076639175415, 1.0928508043289185, -0.4854474663734436, 0.6422044038772583, -0.8099012970924377, 0.20008043944835663, -0.6030075550079346, -0.11302492022514343, -1.1178200244903564, -0.9490301609039307, -1.6220052242279053, -0.5778638124465942, 0.0208777766674757, -0.3951593339443207, 0.6082813143730164, 0.38085034489631653, 1.0370017290115356, 0.4799509048461914, -0.9283266663551331, 0.09818068891763687, -0.24699446558952332, 0.9600831270217896, -0.7090691328048706, -0.6282224059104919, -1.3057864904403687, -1.0073859691619873, -0.5203007459640503, -0.6758904457092285, 1.1794816255569458, 1.519784688949585, 0.4897006154060364, -0.9480928182601929, -1.548204779624939, -0.42118746042251587, 1.5827628374099731, 0.37386801838874817, 0.1747925579547882, 0.8563616871833801, -0.7524288892745972, 0.04748687148094177, 0.5435047745704651, 1.1703381538391113, -0.014162297360599041, -0.6470508575439453, 1.2886855602264404, 2.6851248741149902, 0.6546337008476257, -2.3794214725494385, 2.211298942565918, -0.8010909557342529, 0.9493356943130493, 0.21615152060985565, 0.12261901795864105, 0.19061753153800964, -1.3136675357818604, 1.2226969003677368, -0.8451312780380249, 0.828135073184967, 0.05488881096243858, 0.7167903184890747, 0.08486396819353104, -1.5072938203811646, 1.5409045219421387, 0.7660509943962097, 0.4483063220977783, 0.6016902327537537, -1.5584619045257568, 0.9736907482147217, -0.8054298758506775, -0.5919888019561768], [0.018010899424552917, -0.34224390983581543, 0.5328109860420227, -1.4363195896148682, 2.309537410736084, 0.46183907985687256, 0.6113755702972412, 1.2918552160263062, 1.5878546237945557, 0.37823835015296936, 0.5771960020065308, -0.727795422077179, 1.4842637777328491, 0.2704964578151703, -1.9460676908493042, -0.5494303107261658, -1.5586278438568115, -2.1593728065490723, 1.077150821685791, -0.3955468535423279, -0.46992695331573486, 0.36774152517318726, -1.790350079536438, -0.2991555631160736, -0.39378228783607483, 0.6527962684631348, 0.282532662153244, -1.4914486408233643, 0.3036338686943054, -0.9401665329933167, 0.20087316632270813, -1.3352866172790527, 0.15404638648033142, -0.13381902873516083, 0.15988720953464508, -0.771580159664154, 1.3452847003936768, -0.6010440587997437, -0.7823401093482971, 1.2839384078979492, 0.3914244472980499, 0.9449812769889832, 0.06404981017112732, -1.5949829816818237, -1.3635988235473633, 0.10366271436214447, -0.837889552116394, -2.8040196895599365, -2.4373421669006348, -1.8691514730453491, -1.5531635284423828, -1.9103931188583374, 0.6393008232116699, 0.28218138217926025, 0.1792791187763214, 0.910283088684082, 0.33027008175849915, -0.11993958801031113, 0.6370536088943481, -0.2575605511665344, -1.784145474433899, -0.11641636490821838, 0.0612495020031929, -1.2815821170806885, -0.715828537940979, 0.21556688845157623, -0.6417421102523804, 0.9016455411911011, -1.7979146242141724, -0.7642856240272522, 0.42314648628234863, -2.5392282009124756, -0.27432504296302795, -0.10652422159910202, -1.629648208618164, 0.47570401430130005, -0.06905718147754669, -0.3889872133731842, -0.6779941320419312, 0.5496000051498413, -0.3419061005115509, -0.27458712458610535, -0.026922304183244705, 0.7576997876167297, 1.9106881618499756, -0.7589210271835327, -0.8352739810943604, 0.46210500597953796, 0.8982295393943787, 0.5257142782211304, 1.5379270315170288, -0.2660980522632599, 0.05791591852903366, 1.063094139099121, -0.4883732199668884, -0.8685011267662048, 0.029561365023255348, 0.41821783781051636, 3.572566270828247, 0.3041384816169739, 1.1097184419631958, 0.7196033596992493, 1.1208906173706055, 0.29603591561317444, -0.7728869915008545, -0.1404029130935669, -2.6502435207366943, 1.4208959341049194, 0.19281432032585144, 0.16720078885555267, -0.4062815010547638, 0.3666817545890808, -0.9008250832557678, -0.0482209138572216, -0.6472979784011841, 0.2005627602338791, 0.6387302279472351, -0.5625503063201904, -1.381476640701294, 0.957680881023407, 0.11329075694084167, -0.6305372714996338, -0.6431508660316467, -2.480588912963867, -0.32820427417755127, -0.46346616744995117, 0.5170614123344421, 1.8122652769088745], [-1.1528499126434326, -1.411811351776123, 1.2093532085418701, -0.6741794347763062, -0.7495042085647583, -0.33908435702323914, -0.12282104790210724, 1.6816679239273071, 0.6849623322486877, 1.013077974319458, -0.8141584396362305, -0.05298713594675064, -1.8468223810195923, 1.7160844802856445, 1.3240232467651367, -1.2677069902420044, -0.12484761327505112, -0.9102590680122375, 1.8038101196289062, 2.2592029571533203, -1.2498403787612915, -0.5801215767860413, 1.2999402284622192, -0.3564779460430145, -0.8532077074050903, -3.320289373397827, -0.9128968119621277, 1.0907610654830933, -0.3069359064102173, 0.3379422724246979, -0.41872721910476685, 0.7341175675392151, 1.4564775228500366, 0.7635170817375183, -1.4791597127914429, -0.13923871517181396, -1.6517659425735474, -0.4455457031726837, -1.0062462091445923, -1.1347626447677612, 1.1917593479156494, 0.49531370401382446, 1.540221929550171, 0.1495605856180191, -1.3192647695541382, 1.3340375423431396, -0.30409887433052063, -0.7061998844146729, 1.343515157699585, -0.08195508271455765, 0.11391457915306091, -1.7381311655044556, 0.1950892210006714, -0.38246604800224304, -2.269725799560547, 0.5044454336166382, -0.03698413074016571, -0.012607095763087273, -0.06558732688426971, 0.18557697534561157, -1.8475031852722168, -0.4719860255718231, -0.786945641040802, -0.537203311920166, -0.4877378046512604, 0.14465469121932983, 0.5566456913948059, 0.3329131603240967, 2.015498161315918, 2.4096925258636475, 0.8274649977684021, 1.0939092636108398, -0.1679486334323883, 0.13960422575473785, -0.5581537485122681, 0.036178432404994965, -1.036972165107727, 0.04182463511824608, 1.5597018003463745, 0.7462849020957947, 1.5629714727401733, -0.1175791472196579, -0.1896049827337265, -0.6457514762878418, -1.304153323173523, 0.43578195571899414, -1.918274998664856, -1.2311745882034302, 0.9927929043769836, -0.2665417492389679, 0.3850838541984558, 1.0214455127716064, 0.8848719596862793, -0.23022392392158508, 0.41492730379104614, -1.5819517374038696, -0.7756546139717102, 1.9544047117233276, -0.1939634084701538, -1.2075318098068237, -0.8704164028167725, -1.7963072061538696, -0.5692726373672485, 2.456510066986084, 1.111270785331726, -0.9030163288116455, 0.03651324287056923, 2.12591552734375, 0.37599098682403564, 2.038048028945923, -0.37770944833755493, -1.0919543504714966, -0.6845356822013855, 1.0534896850585938, -1.6750974655151367, 1.304257869720459, 0.7385644316673279, 0.3547057807445526, -0.26225975155830383, 0.600619375705719, 0.775225043296814, 1.38548743724823, 0.7293084263801575, 0.9337021708488464, 1.658394694328308, -1.3823732137680054, 2.771230936050415, -0.6914165019989014], [-0.12681706249713898, 0.7279068231582642, -0.6914908289909363, -0.46473440527915955, -0.9764381051063538, -0.5715213418006897, -2.689220428466797, -0.08710449188947678, -0.5954190492630005, 0.5459666848182678, -0.25009945034980774, -2.1983144283294678, 2.3635268211364746, -0.2583036720752716, 0.9730275273323059, 0.019387638196349144, 0.2892420291900635, -1.6086400747299194, -0.24113453924655914, 1.3626080751419067, 0.3289034068584442, -0.9624720215797424, -0.2938535809516907, -0.798581600189209, -0.5127507448196411, 0.5186240077018738, 2.068978786468506, 0.2615514099597931, -0.48537346720695496, -0.1777779757976532, 0.4363507628440857, -0.6266314387321472, -1.4308466911315918, -0.3453153967857361, 0.29413166642189026, -0.7165011763572693, 0.2584981322288513, 2.028459310531616, -1.0152149200439453, -0.47348129749298096, 0.6263444423675537, 1.709251880645752, 2.0121536254882812, -1.0108981132507324, 0.8951091170310974, -0.17845045030117035, -0.07555214315652847, 0.4839267134666443, -2.076148509979248, 0.09071176499128342, -1.2679097652435303, 0.3072340488433838, -0.8866868615150452, -0.48871901631355286, 0.3380625545978546, 0.2147504836320877, 0.18423223495483398, 0.9387712478637695, 1.3375760316848755, 1.1890393495559692, -0.6310360431671143, 0.7661644220352173, 0.0988561138510704, 1.1350514888763428, -2.0322043895721436, -0.441508948802948, 0.7127591967582703, 1.2636638879776, 1.9791537523269653, 0.11232469975948334, 1.4787602424621582, -1.124474287033081, 0.2617306411266327, -0.4637377858161926, -0.9481508731842041, -0.050383489578962326, -0.3940250277519226, 0.90362948179245, -1.6354933977127075, 0.4970889091491699, 1.9000065326690674, -0.9906184077262878, -0.30533865094184875, -0.7834120392799377, 1.3031941652297974, -0.672453761100769, -0.17360371351242065, 1.719253420829773, 0.9742052555084229, -0.7383782863616943, -0.03758695349097252, 0.8118271827697754, -0.7068163156509399, -1.2412803173065186, -0.4533597230911255, 0.7208154797554016, -1.287535548210144, -0.4620665907859802, -2.3369827270507812, -0.21601402759552002, -1.4691882133483887, -0.019563427194952965, 0.915986955165863, -0.2818189561367035, -0.5720125436782837, 1.1279405355453491, -0.9720061421394348, 0.4726954996585846, -1.8857427835464478, -0.9662348031997681, -0.2923581302165985, -0.4423416554927826, -0.4033842980861664, -2.1095523834228516, 0.42230474948883057, 0.13189291954040527, 1.2925612926483154, 0.6062468886375427, -1.4743866920471191, 1.0801379680633545, 0.7481030821800232, 1.2272087335586548, -0.6864854693412781, -1.4124386310577393, 0.21854408085346222, -0.16133631765842438, -0.050156861543655396, -1.5606293678283691], [-1.8209178447723389, -1.50877046585083, 1.3131619691848755, 0.4631865918636322, -0.9977114796638489, 2.519200563430786, 0.14373888075351715, 0.13752244412899017, -1.7218588590621948, -1.1948778629302979, -0.957884669303894, -1.2053805589675903, -0.6215386986732483, 1.190258502960205, 0.790820300579071, -0.9955974221229553, 0.4070607125759125, 0.1295388787984848, 0.7171757221221924, -0.25520437955856323, 1.742403507232666, -0.537855327129364, -0.6965174674987793, 1.0151935815811157, -0.6987254023551941, 0.6257919073104858, -0.8077862858772278, -0.2745034098625183, 0.3746844232082367, -0.49037837982177734, -0.7003166079521179, -2.0194287300109863, 1.0852165222167969, -0.7954407930374146, -1.1550171375274658, -0.2924720048904419, 0.2015388309955597, 0.03876276686787605, 1.314696192741394, 1.6502363681793213, -2.176787853240967, -0.1978577971458435, -0.4202231466770172, 0.03239153325557709, -1.3407105207443237, 0.4342557191848755, 1.338306188583374, -0.3864499032497406, -0.2055668830871582, 0.3588240444660187, -0.20184564590454102, -0.12537835538387299, -1.704649567604065, 1.0950013399124146, 0.921394407749176, 0.12447631359100342, -0.12706759572029114, -1.7899808883666992, -2.804837942123413, 0.058100685477256775, -1.5758917331695557, -0.7806314826011658, 0.1251048594713211, 1.558593988418579, -1.6055738925933838, -0.009492514654994011, 0.8934255838394165, 3.8382394313812256, -1.0710068941116333, -0.3562861680984497, -0.8958008885383606, 1.0724678039550781, -0.7980605363845825, -0.6798520088195801, 0.06692753732204437, -0.501853883266449, -0.6013516783714294, -1.023325800895691, -1.7933751344680786, -0.5657215118408203, 0.39150407910346985, -1.5190613269805908, -0.25377076864242554, -0.5238391757011414, 0.03726811334490776, 0.9964103698730469, -0.5977025032043457, 1.2664226293563843, 0.02512321248650551, -1.6846439838409424, 0.47385644912719727, -0.9645575881004333, -0.4604126214981079, 1.836955189704895, 0.2544618844985962, -1.3135613203048706, 0.42586377263069153, 2.1844491958618164, 0.807424783706665, -1.208022952079773, -0.9719500541687012, 0.6414309144020081, -1.7691744565963745, 1.6533770561218262, 0.39747491478919983, 2.806809663772583, 0.962909996509552, -2.0396976470947266, -0.06315532326698303, 0.9271148443222046, -0.6689480543136597, -0.6230803728103638, 0.14749784767627716, -0.5268746614456177, -0.583829402923584, -1.5582942962646484, 1.314744234085083, -0.1195530891418457, -0.828899085521698, -1.46846604347229, -0.4939177930355072, -0.7406464219093323, 0.726701021194458, 0.03508612886071205, -0.910507321357727, -0.09061520546674728, 1.8571773767471313, 1.7535845041275024], [-0.6917139291763306, -1.6546765565872192, 0.04949484393000603, 1.0612432956695557, 1.9088910818099976, -0.6584452390670776, -0.22628305852413177, -0.03958745300769806, 1.467577338218689, -1.3817843198776245, -0.42733657360076904, 0.6598827242851257, -0.7489652037620544, -0.6431863307952881, -0.14791424572467804, -0.3787541687488556, -0.33384814858436584, -1.002510666847229, -0.005065153352916241, -0.43954038619995117, -0.05140930041670799, -0.023653047159314156, -0.028647001832723618, -0.37944841384887695, 0.36907434463500977, 0.4875796437263489, 1.6235992908477783, 0.9167732000350952, -2.7588987350463867, -1.6602662801742554, 0.8710394501686096, 2.3470993041992188, 0.01571086049079895, 0.7656692862510681, -1.6262564659118652, 0.6331489682197571, 0.004957945551723242, 0.7538238763809204, -0.03478005900979042, -0.2785907983779907, -0.3712213933467865, -0.7110332250595093, -0.7501758933067322, -0.0534224808216095, 0.49052664637565613, 2.5298190116882324, 0.2971828281879425, 0.8976858258247375, 1.2540432214736938, -1.0191905498504639, 0.3752223551273346, 0.38560202717781067, -2.4994170665740967, 2.598726749420166, -2.5117273330688477, -0.5583332180976868, -1.6901586055755615, 0.6752133369445801, 0.928987443447113, 0.1919766366481781, 1.7097054719924927, 0.5299773812294006, 0.6946293711662292, -1.8636291027069092, 0.2837240397930145, 0.7522716522216797, -0.9410545229911804, -0.538766622543335, -0.10934364795684814, -0.5194687247276306, 1.0153828859329224, -0.24953287839889526, 0.21523864567279816, 0.560248851776123, -0.04352631792426109, 0.7297568321228027, 0.7823652625083923, -0.6961772441864014, 0.9389952421188354, -0.17070718109607697, 0.631708562374115, 0.12379290163516998, -1.0583114624023438, -0.1281697303056717, -0.38964083790779114, -0.687044084072113, -0.9611250162124634, 1.3273193836212158, 0.8175681233406067, 0.8750435709953308, 0.32711565494537354, -0.16933801770210266, -0.3312912583351135, -0.45618125796318054, 1.5054253339767456, -0.48036545515060425, -0.7746046781539917, -1.2143137454986572, 0.3574486970901489, -0.9791609048843384, -1.6126512289047241, 0.046639375388622284, -1.0851057767868042, -0.641559898853302, 0.04314907267689705, 1.0549286603927612, 0.7001598477363586, -0.33169299364089966, -0.9882423281669617, -2.622713565826416, -0.18685726821422577, 0.3989599943161011, 0.23757559061050415, 0.06192982196807861, 0.09809314459562302, 0.37324216961860657, -0.24754808843135834, -0.5070050358772278, -0.8607664108276367, -0.6294440627098083, 0.02772885374724865, 0.6739934086799622, -1.0750523805618286, -1.2967760562896729, -0.08509865403175354, -1.1344376802444458, -0.5103552341461182, -1.0209630727767944], [0.4559413492679596, -0.9838285446166992, 0.1753593236207962, 0.2900483310222626, -1.6159205436706543, -0.11357876658439636, 0.3732747733592987, 1.0587830543518066, -1.5802083015441895, 0.6179983615875244, 0.04440653696656227, -1.36572265625, 0.1961621791124344, 1.3925198316574097, 1.5476386547088623, 1.9967248439788818, -0.16496479511260986, 0.8619353771209717, -0.29698750376701355, 1.0878558158874512, -0.5265936851501465, -0.07764964550733566, 0.5413728952407837, 1.6240439414978027, 0.10295998305082321, -0.7606860399246216, -0.27438783645629883, 0.16469688713550568, 1.4108115434646606, -1.3483434915542603, 0.8200547695159912, 0.5481754541397095, -0.46230193972587585, 0.2999640703201294, 0.679972767829895, -0.9351767897605896, -0.5164346098899841, 0.03972591459751129, -0.2461184412240982, -0.11764821410179138, 0.41995999217033386, 0.38147658109664917, 0.5654390454292297, 0.14591148495674133, 0.43269044160842896, -2.0106964111328125, -0.7075744867324829, 0.5503678917884827, -0.9975140690803528, -0.5024551749229431, 0.3944331705570221, -1.846573829650879, -0.1435314118862152, -0.5971802473068237, 0.9560595750808716, 0.3676075041294098, -0.6512596607208252, 1.0463387966156006, -0.173463374376297, 0.05644708871841431, -0.9586559534072876, -0.05989587679505348, 0.3548028767108917, 2.009955883026123, 0.23583155870437622, -0.7916621565818787, -1.1256515979766846, -0.450040340423584, -0.450611412525177, -0.38503655791282654, -0.4970298409461975, 0.64311683177948, 0.5857415199279785, -0.41219326853752136, 0.7262128591537476, -0.3046438992023468, -0.3523974120616913, 0.7737962603569031, -0.42824044823646545, -0.9267284274101257, 0.19597232341766357, 0.22333796322345734, -0.9649034738540649, 0.9690079689025879, 1.4346182346343994, 0.7580299377441406, 1.0453541278839111, 0.9826677441596985, 0.10192494839429855, 0.009440433233976364, -1.9276888370513916, -3.278179883956909, -0.12839248776435852, -0.8153926134109497, 0.7046979665756226, 0.21564064919948578, -0.19123254716396332, 0.2952454686164856, 1.5132914781570435, -0.2559509575366974, 0.1917746365070343, 0.6561054587364197, 0.11010868847370148, -1.4151203632354736, 0.5853778123855591, 1.7411924600601196, -1.0563571453094482, 0.42542770504951477, -0.2639341652393341, 0.20257608592510223, -2.356050491333008, 0.897567093372345, -0.8319637179374695, 0.2197471708059311, -0.6041406393051147, 0.705793023109436, -0.02793513797223568, -1.0525767803192139, -0.9297210574150085, -2.360827922821045, 0.056234173476696014, 0.3704947531223297, 0.38860660791397095, -0.5559933185577393, -0.22395917773246765, 0.6931819915771484, 1.5691359043121338, -0.43105950951576233], [0.45888185501098633, -0.9166057705879211, -0.259651780128479, -0.9186668992042542, 1.645545482635498, 0.6624637246131897, 0.07414248585700989, -1.0948010683059692, 0.31406348943710327, -0.2242925465106964, 0.6515798568725586, 0.15206493437290192, -0.12077146768569946, 0.18132880330085754, 0.7218031883239746, 0.1435346007347107, -0.7147091031074524, -1.3775110244750977, 0.36329278349876404, -0.8149431943893433, -1.3519266843795776, 1.199825644493103, -1.160402774810791, 0.38629207015037537, 0.996593713760376, -0.09268838912248611, -1.2147425413131714, -0.5396618843078613, -0.1575428545475006, 2.1013917922973633, -1.2144464254379272, 0.1855410933494568, -0.19266727566719055, -0.7671384811401367, -0.31558719277381897, -0.5311905741691589, -0.686180055141449, 0.3338153660297394, -0.3357202708721161, -0.5566315054893494, -0.2701888084411621, 1.9316761493682861, 1.8231807947158813, 1.3106039762496948, -2.0279974937438965, 0.32775866985321045, -0.6215399503707886, 0.7459283471107483, -1.5306816101074219, -0.6703010201454163, -0.47002550959587097, 0.3082987368106842, 1.0773403644561768, -0.593979001045227, -0.43757933378219604, 1.238554835319519, 0.3703315556049347, 0.9981800317764282, -0.41405054926872253, 0.36470845341682434, -0.17747358977794647, -0.6471892595291138, 2.9992945194244385, -0.2269740253686905, 1.8187470436096191, 0.026279177516698837, -0.13257184624671936, -0.7470633387565613, -0.403065025806427, -0.5580453276634216, 2.8212270736694336, -1.1568750143051147, 0.4713656008243561, 0.977130115032196, 0.8465715050697327, 2.485597610473633, 1.5438603162765503, 0.8165083527565002, 1.581009864807129, 0.3573165833950043, 1.2786017656326294, 0.6150640845298767, -1.7141475677490234, -0.4977400004863739, 1.7888789176940918, -0.3140660226345062, -0.2130449265241623, 0.5167763233184814, -2.272049903869629, -0.5615922808647156, 0.08180128037929535, -0.3548131287097931, 0.4437432289123535, -0.48789259791374207, 0.7265754342079163, -0.07681101560592651, -1.4760229587554932, 0.7313848733901978, -1.2528187036514282, 1.621501088142395, 0.5067312717437744, 0.19228523969650269, 0.2393294721841812, -0.15045545995235443, 0.817139744758606, -2.02463698387146, -0.3222767114639282, -1.1466279029846191, -2.152791976928711, -1.6526620388031006, 0.5496074557304382, 0.03182917460799217, -0.26171159744262695, -0.4921361207962036, -0.9483136534690857, 0.09159788489341736, -0.22640635073184967, 1.02601158618927, -0.10358280688524246, -1.017577052116394, -0.7866242527961731, -0.011473606340587139, 0.1643245965242386, -0.6756913661956787, -0.22607757151126862, 1.2080588340759277, -0.7666933536529541, -0.796652615070343], [0.37507104873657227, -0.5575903654098511, 0.2543576657772064, 0.9822249412536621, 0.14251570403575897, 1.541061282157898, 2.1448497772216797, 2.4296839237213135, 0.6779826879501343, 1.321622371673584, 0.30112630128860474, -0.41198965907096863, -1.5232102870941162, 0.7446689009666443, -0.7734702825546265, 0.3920547366142273, 0.043067753314971924, 0.710188627243042, 0.027991198003292084, 1.19808030128479, -0.18221139907836914, -0.9536191821098328, 0.3764960765838623, 1.3600494861602783, 0.5088561177253723, -0.17307494580745697, 1.6030105352401733, 0.6451241970062256, 1.527985692024231, 1.1100053787231445, -0.49342650175094604, -0.21453969180583954, -1.5819675922393799, 1.277125358581543, 2.036181688308716, -0.043417006731033325, -2.0130462646484375, -0.835511326789856, 1.4660638570785522, -0.831935703754425, 0.9508746862411499, -0.1863652914762497, -0.4632752537727356, 0.6176808476448059, 2.0986111164093018, 1.7467641830444336, -1.9374738931655884, 0.44235846400260925, 0.16121385991573334, 0.33479762077331543, -0.6439937353134155, 1.199906587600708, 1.0844051837921143, 0.19899000227451324, 1.2439920902252197, 1.4691038131713867, -0.28011807799339294, 0.1118539571762085, -1.4193153381347656, -1.1138194799423218, 1.6477646827697754, 0.6021167039871216, 0.37966033816337585, -2.466918468475342, -1.3461893796920776, -0.8348926901817322, -1.4116700887680054, 1.110548734664917, -0.6046148538589478, 1.1800744533538818, 0.5274761319160461, -0.832129180431366, 0.07982544600963593, -0.9092229604721069, 0.18889719247817993, 2.208096504211426, -0.6259039640426636, 0.8056881427764893, -2.0809760093688965, 0.1123710498213768, 0.24710002541542053, 0.9283009767532349, -0.027578938752412796, 0.7845785021781921, -0.18340450525283813, -0.6632095575332642, 0.5460186004638672, 0.99764484167099, -1.4922343492507935, -2.1788387298583984, -0.021662039682269096, -2.4887776374816895, 0.6252923011779785, -1.7388235330581665, 0.6545332074165344, 0.5691698789596558, 0.5518009662628174, 0.22817747294902802, -0.39509978890419006, 0.32356691360473633, 0.5968625545501709, 1.1949009895324707, 1.49613356590271, -0.9400961995124817, -0.49417704343795776, 0.18688036501407623, -1.3844952583312988, 0.7040742635726929, -0.38862094283103943, 1.0497866868972778, -0.030957812443375587, 1.536137580871582, 0.17073312401771545, 0.44698721170425415, -0.994929313659668, -0.028488771989941597, -0.25166642665863037, -1.384432315826416, -0.327210932970047, -1.2173594236373901, -0.0041779703460633755, -0.701150119304657, -1.6361302137374878, 0.9653045535087585, -1.513501524925232, 1.0739444494247437, -1.3782029151916504, 1.146426796913147], [0.4276304543018341, 2.2411251068115234, 0.07307619601488113, 0.18927721679210663, -0.5185310244560242, 1.6350244283676147, -0.6162227392196655, -1.2650153636932373, -0.8700589537620544, 2.120802402496338, 1.2249736785888672, 1.1995373964309692, 0.20955874025821686, 0.13605941832065582, -0.7198922038078308, 2.043316602706909, -1.0183466672897339, -0.42668789625167847, -0.3133086860179901, -0.17388425767421722, 0.629813015460968, -1.71226966381073, 1.2851203680038452, -0.8020821213722229, 0.36328160762786865, 1.1894501447677612, 1.2896941900253296, -3.202195644378662, -0.21593959629535675, 0.7194749116897583, -0.4164152145385742, 0.14492729306221008, 0.6365329623222351, -0.741416335105896, -0.48996660113334656, -0.4940235912799835, 0.33602380752563477, -0.4299164116382599, -0.10617250949144363, 0.6669667363166809, -0.41912946105003357, -0.5733587741851807, -1.2644925117492676, -1.161959171295166, -0.9254671335220337, 0.9197455644607544, 1.998971939086914, 0.8579639196395874, 0.10719955712556839, 0.11111552268266678, -1.3763874769210815, -0.008847968652844429, -0.2051994651556015, 0.72513747215271, -0.5707945227622986, -1.1785215139389038, -0.5781501531600952, 0.9810057282447815, -0.806219220161438, 0.15590710937976837, 1.1015795469284058, -0.46939048171043396, 0.7261969447135925, -0.48853495717048645, 0.127830371260643, -0.5215618014335632, -1.000813364982605, -0.33671653270721436, 0.13661253452301025, -0.8971675634384155, -0.22585135698318481, -0.39165860414505005, -0.8816002011299133, 1.093934416770935, -0.9095534682273865, -0.42347678542137146, 0.34876757860183716, -1.1514519453048706, -1.2020103931427002, 1.1072518825531006, -0.040267642587423325, 1.9696919918060303, 0.9594875574111938, -0.1600898951292038, 0.5924155116081238, 0.296699583530426, 2.046013832092285, -0.4568999409675598, 0.3227784037590027, -1.9331920146942139, 0.22619450092315674, -2.332714557647705, -0.16200542449951172, -0.29160434007644653, 0.21024635434150696, -0.3770304024219513, 0.20330923795700073, -2.114656686782837, -1.7504254579544067, -0.06645438820123672, 0.7002445459365845, -0.1561318039894104, -1.0447298288345337, 0.45806044340133667, 0.574874222278595, -0.5151743292808533, -0.2746405303478241, -1.5221209526062012, -0.7004665732383728, 2.0357487201690674, 1.0797407627105713, 1.8532198667526245, -0.3406098783016205, 0.6413727402687073, -0.40957942605018616, -0.9287768006324768, 2.9322991371154785, -0.9276764392852783, 0.7721478939056396, 1.1332907676696777, -0.26632943749427795, -0.42006567120552063, 0.6463987827301025, 0.589766263961792, -1.1022167205810547, -0.7729964852333069, -0.17714980244636536, -0.5274418592453003], [1.5184181928634644, 0.013649996370077133, -1.3810834884643555, 1.0933490991592407, -1.3282430171966553, -2.0786056518554688, -0.7091884016990662, 0.6081928610801697, 1.2007375955581665, -0.09126638621091843, -1.3507144451141357, 0.06128319352865219, 0.22327707707881927, 1.0751293897628784, -1.5794174671173096, -0.3678199052810669, 0.7581079006195068, 0.009969878941774368, -0.30201488733291626, 0.6567955613136292, -0.7657263278961182, 1.399207592010498, -0.2650204002857208, 1.8381913900375366, 1.136672019958496, 0.011590211652219296, -0.392463356256485, 1.535001277923584, -2.5007271766662598, -0.5663425326347351, -0.3433425724506378, 0.921164870262146, 1.3580557107925415, -0.20004235208034515, 0.24908941984176636, -0.5384416580200195, -2.8948519229888916, 0.6997748017311096, 0.2271386831998825, 0.8971970677375793, -0.8357171416282654, 1.0948917865753174, 0.8910509943962097, -0.3370833396911621, -0.1741214245557785, -0.943720281124115, -1.4758719205856323, 0.4622449278831482, 0.30059269070625305, -0.9856783151626587, -2.184380292892456, 1.331681251525879, -0.5941256284713745, -1.008087158203125, 1.3100390434265137, 1.1401253938674927, 0.5267931222915649, -0.41643962264060974, 1.1254503726959229, 0.8491217494010925, 2.1344356536865234, 1.7060242891311646, 1.6114957332611084, 0.5758430361747742, -1.2041493654251099, 1.71964693069458, -0.9511248469352722, 1.2710204124450684, -0.9627959728240967, -3.1009819507598877, -1.5140960216522217, -2.007171154022217, -1.26082444190979, -0.8175815939903259, -2.5163824558258057, -1.6057898998260498, -0.12488431483507156, 1.5621225833892822, -1.7390592098236084, 1.3906162977218628, -0.28024619817733765, -0.7587875723838806, -1.3311889171600342, 0.8015836477279663, -0.22642628848552704, -0.3011776804924011, -0.9697802066802979, -0.04048847034573555, -2.0091381072998047, -0.7109588384628296, 1.2296441793441772, 0.4919637441635132, -0.8499639630317688, -1.1680670976638794, 0.8500310182571411, -0.33746373653411865, -2.1896450519561768, -0.836771547794342, 0.6182937622070312, -0.09112361073493958, -0.665584921836853, 0.9412921071052551, -0.4572061002254486, -1.6735835075378418, 1.0239806175231934, -1.0524401664733887, -0.8399176001548767, 0.6891996264457703, -0.4247450828552246, 1.7993366718292236, -1.401789903640747, 0.606042206287384, 1.739983320236206, 0.08216076344251633, -1.1548995971679688, 0.3073860704898834, -1.7265384197235107, 0.33136439323425293, 1.918932557106018, 0.8495914340019226, 0.6497382521629333, 1.5556656122207642, 0.27669888734817505, 0.14634324610233307, 2.268918991088867, 0.11263308674097061, 1.5900248289108276, -0.7383838891983032], [0.6480459570884705, 0.4207637906074524, -0.8888060450553894, -0.06272489577531815, -1.4823887348175049, 0.45355966687202454, 0.10762803256511688, 0.5687398910522461, 0.942867636680603, -1.9600638151168823, -1.1643280982971191, -0.9723634719848633, 0.11765103787183762, -0.6491073966026306, 0.7643249034881592, 0.03712427616119385, 1.0847687721252441, 0.034003596752882004, 1.01058828830719, 1.6981277465820312, -2.402320146560669, -0.5911250710487366, -0.026828087866306305, -1.5848294496536255, -0.7040013074874878, -0.011687920428812504, 0.210536926984787, 0.6246543526649475, -0.7152550220489502, -0.6951094269752502, 0.9973257780075073, 0.6025581359863281, 0.8558825254440308, -0.3364289104938507, 0.6972851157188416, -0.810609757900238, 0.0494236946105957, 1.46388578414917, -0.28272369503974915, 1.9605659246444702, -0.7543878555297852, 1.4688936471939087, -1.4812530279159546, -0.7647476196289062, 1.1743528842926025, 0.4191935062408447, 1.5957385301589966, 0.5247409343719482, -0.6659165620803833, -0.03132129833102226, -0.06931692361831665, -0.001686077332124114, 0.2879554033279419, -0.0796789601445198, 0.9271527528762817, 0.694789707660675, -0.44709911942481995, -0.809539258480072, 0.2865549623966217, 1.3587044477462769, 1.8136231899261475, -1.4823118448257446, 0.01725110225379467, 0.18181677162647247, 2.4277100563049316, -0.8471546769142151, -0.29962849617004395, 0.4569960832595825, 0.8499102592468262, 0.7701585292816162, -0.212073415517807, -0.5204710364341736, 1.327393889427185, 0.9241991639137268, -0.01635017618536949, 1.1417653560638428, -1.1143547296524048, -0.6270795464515686, -0.3389154076576233, 1.7490183115005493, -0.7463890910148621, 2.422922134399414, 1.3103554248809814, 0.5354172587394714, 0.34052515029907227, -0.21459028124809265, 0.8428077101707458, -2.5005548000335693, 0.08953089267015457, 0.06586681306362152, 0.7116402387619019, 0.7122110724449158, -0.927066445350647, -0.1641012281179428, -0.5672250390052795, -0.4206331968307495, 0.10242359340190887, 0.6964182257652283, -1.232853651046753, 1.5154131650924683, -0.5030217170715332, -0.47589078545570374, -0.18334554135799408, -1.3648103475570679, 0.16091656684875488, -0.00033054931554943323, 0.4611778259277344, 0.06709592789411545, 0.7581786513328552, -0.4201551377773285, 1.2969034910202026, -1.1509679555892944, 0.5477117300033569, 0.8546593189239502, 0.5478782653808594, 0.7256967425346375, -0.679012656211853, -0.7138568758964539, -1.9143000841140747, 0.9012520909309387, 1.3546208143234253, 0.03885743394494057, -0.014536898583173752, 0.6804400086402893, -2.7552459239959717, -0.24757643043994904, 0.49133217334747314, 0.9685760140419006], [-0.15018759667873383, -0.16753050684928894, -0.4604560136795044, -1.183682918548584, -0.9692088961601257, 0.5584844946861267, -0.7986268997192383, -0.6060567498207092, 0.36698150634765625, -1.0350102186203003, -0.7605183124542236, 0.6212794780731201, -0.1007252186536789, -0.500656247138977, -0.7255682349205017, 0.9733703136444092, 0.15387119352817535, -0.5964972376823425, -0.766287088394165, -1.2944473028182983, 0.08565602451562881, -0.197846457362175, -0.0833326056599617, 1.0723600387573242, 0.4644457995891571, -0.5166462659835815, -1.1453887224197388, 1.5388387441635132, -1.8249461650848389, -1.9253698587417603, -0.20442958176136017, 1.057421326637268, 0.9078757166862488, 0.8829938173294067, 0.9718453884124756, -0.650105893611908, -0.18623673915863037, 1.0889812707901, 1.2316100597381592, -1.3468838930130005, 0.014727510511875153, -0.4617830812931061, -0.2985322177410126, 1.3661949634552002, -0.787372887134552, 0.6827859282493591, -1.8983123302459717, 0.5024586319923401, -1.98638117313385, 0.09494256228208542, 0.374401330947876, 1.6673091650009155, -0.3419862687587738, 0.4649469256401062, 0.988544762134552, 0.8246683478355408, -0.20854882895946503, -2.1266379356384277, 0.24600987136363983, 0.435092568397522, -1.039416790008545, 2.4650728702545166, 0.92145174741745, 1.0922967195510864, 1.608741044998169, -0.46719545125961304, 1.3236087560653687, 0.8740896582603455, -1.0112810134887695, 1.0963462591171265, -0.486910343170166, 1.4702060222625732, -0.4960739016532898, 0.2266533225774765, 0.6741242408752441, -0.1254887878894806, -1.0490806102752686, 0.36953145265579224, 0.9255936741828918, -1.5613744258880615, 1.670446753501892, 0.00271525583229959, -1.4446297883987427, -2.1428494453430176, -0.4433177709579468, -0.6433059573173523, -1.2156628370285034, 0.5940017700195312, -0.4801773428916931, -0.3285861611366272, -0.08275307714939117, -0.170780748128891, -0.30225515365600586, 0.1776556819677353, 0.2913203537464142, -1.9706848859786987, -1.2904574871063232, 0.7489926815032959, 0.9673711061477661, -0.5970538258552551, 1.2241092920303345, -1.417694330215454, -0.6813344955444336, 0.2434110939502716, -0.9950539469718933, -0.14907582104206085, 0.5511752963066101, -2.1111185550689697, -0.990155816078186, -1.1833930015563965, 0.6329793930053711, 1.0749328136444092, 0.9341970682144165, 0.9567198753356934, 0.2925699055194855, -0.27938583493232727, -1.4082478284835815, -0.3028646409511566, 0.03455105796456337, 1.0791511535644531, 0.42858636379241943, 1.088240146636963, -0.9533748626708984, 2.0294578075408936, -0.3848171830177307, -0.4543798267841339, -0.568855345249176, 0.0908694788813591], [-1.009390950202942, 1.9177212715148926, -0.3608816862106323, 0.9572559595108032, 0.7064520716667175, -1.5795272588729858, 0.134871244430542, 1.0170267820358276, -0.9993326663970947, -1.2216532230377197, -0.2034652978181839, 0.5287652015686035, 0.581272304058075, 0.2653510570526123, 0.04175630956888199, -0.16247357428073883, -0.8013140559196472, -0.06814105808734894, -3.373800754547119, 1.655347228050232, -0.8922683000564575, -1.7924081087112427, -0.3541494607925415, -1.963432788848877, -0.760485827922821, 0.4728558361530304, -0.2730758786201477, -1.6021629571914673, 0.450051873922348, -0.8736509084701538, 1.3811368942260742, -0.25594496726989746, 0.06945676356554031, 0.932619571685791, 0.85390704870224, -1.1597214937210083, -1.4916341304779053, 0.6168299317359924, 1.4050759077072144, -0.19643934071063995, 0.5234128832817078, 0.9293972849845886, -0.24345974624156952, -0.8128155469894409, -0.5973725318908691, 1.011813998222351, 1.1757444143295288, 0.3604982793331146, -0.4186381995677948, 0.9384533166885376, -0.9829778075218201, -0.7631296515464783, 0.6484330892562866, 1.9899355173110962, -0.4289051592350006, -0.0810869038105011, -1.1565438508987427, -0.3450040817260742, -0.8240122199058533, 0.2551468312740326, 0.2898246943950653, 1.1257905960083008, 0.08912960439920425, 0.7914147973060608, 0.39293399453163147, 0.4102966785430908, -0.009494924917817116, -0.8890988826751709, 1.7867375612258911, -0.45908087491989136, -0.676283597946167, -1.3602313995361328, -0.46912840008735657, 0.24699725210666656, 1.607358455657959, 0.18157877027988434, -1.0088039636611938, -0.14452572166919708, 1.1155729293823242, 0.6332852840423584, 1.7520761489868164, -0.6534037590026855, -0.2306099385023117, -0.03916275128722191, 0.8124080896377563, -0.5230833888053894, -0.35739752650260925, 0.034238193184137344, -1.4621962308883667, -1.3529328107833862, -0.0005320595228113234, -0.22761370241641998, -2.005403518676758, 2.946507215499878, -0.8314825892448425, 0.4872698187828064, 0.19945749640464783, -0.5311483144760132, -0.3767673075199127, -0.2986603081226349, -0.08227541297674179, -0.3514638841152191, -0.35028526186943054, -0.20100972056388855, -0.9998093843460083, 0.6341805458068848, 1.0930124521255493, -0.06728202849626541, -0.5269211530685425, 0.2225552201271057, 0.5006804466247559, -0.5788846611976624, -0.228017657995224, 0.386831134557724, -0.0008136165561154485, -0.15422900021076202, 0.16031238436698914, 0.5410416126251221, -0.7497059106826782, -0.7757424116134644, -0.19723843038082123, -0.4050130844116211, 0.020282499492168427, -0.2842918634414673, -1.2846670150756836, -1.7655699253082275, -0.9384196400642395, 0.6017132997512817], [0.5763086676597595, 2.081146717071533, -0.002305626170709729, 0.21400724351406097, 0.8143711090087891, -0.9234182834625244, 0.18018923699855804, -0.7299975156784058, 1.0507956743240356, -0.762755274772644, 0.20505589246749878, 0.4107336401939392, 1.227353811264038, -0.1712092012166977, -0.910411536693573, 1.6382203102111816, -0.3970986306667328, 1.111060619354248, -0.176616370677948, 0.12733429670333862, 1.7879469394683838, 1.369896411895752, 0.5667038559913635, -0.28584814071655273, -1.032207727432251, 1.7906558513641357, -0.15248620510101318, 0.823579728603363, -0.01852218434214592, 0.565943717956543, -0.4175027012825012, -0.6487576961517334, -0.07226549088954926, -0.11976782977581024, -1.0307334661483765, 0.4003485441207886, 1.4225438833236694, 0.413556843996048, 0.1086655780673027, -0.8173801898956299, 1.2699205875396729, -0.38543322682380676, -0.2249172031879425, -1.4327412843704224, 1.066861629486084, 1.324324131011963, 0.5936811566352844, -1.2281299829483032, -0.08239822834730148, -1.3280340433120728, 0.41622814536094666, -0.2713965177536011, -0.8403225541114807, 0.3345043659210205, -0.512198805809021, 1.4635040760040283, -1.6150206327438354, 0.37516549229621887, -0.016680698841810226, 0.8139523267745972, 0.7076894640922546, -0.32034653425216675, -1.859468936920166, -1.7586792707443237, -0.5221039652824402, -0.3781348168849945, 1.5957359075546265, 0.16281886398792267, 0.10566501319408417, 1.1186600923538208, 2.180911064147949, -2.132662773132324, -0.4161512851715088, 0.8768884539604187, -1.1690754890441895, -0.903827428817749, 1.1209572553634644, 0.340503066778183, -0.619638979434967, -0.1829933226108551, -0.07763170450925827, -0.47020915150642395, -0.2413797527551651, -0.24314631521701813, -0.49319714307785034, -0.3267577588558197, 0.6679755449295044, 0.2519983649253845, -0.6357901096343994, 1.5336846113204956, -1.1686402559280396, 1.4819936752319336, 0.875217854976654, -1.037040114402771, -1.362058401107788, -1.0464158058166504, 0.7079319953918457, 0.12176477909088135, 0.9653329253196716, -0.6649217009544373, 0.34240415692329407, 0.19477888941764832, -1.424466848373413, -2.174699544906616, 0.3999175429344177, 0.2454690784215927, 0.17807665467262268, -0.0026454743929207325, 1.8522032499313354, -0.1833847016096115, -0.04512523114681244, 0.42638400197029114, 0.009000588208436966, 0.08019997924566269, 2.352771043777466, -0.3291296064853668, -0.20559343695640564, 1.1692819595336914, -2.5886073112487793, -0.41873928904533386, -0.047985516488552094, -0.4779730439186096, -0.016086077317595482, -0.7526049017906189, 0.08344048261642456, 0.6728997826576233, -1.3166838884353638, 0.07950691133737564], [0.6968739628791809, 1.2221808433532715, 0.4154360890388489, -0.4553939998149872, 0.8504927754402161, -2.3895034790039062, 0.7779264450073242, -1.216303825378418, 0.5611506104469299, -0.4294188320636749, 0.137798011302948, 0.4369318187236786, 0.7720382809638977, -1.2847023010253906, 1.1639689207077026, 0.04354735091328621, -0.5684689283370972, 0.7732250094413757, 0.23211081326007843, 0.4391922950744629, 0.2943064868450165, 1.0662801265716553, 1.0605872869491577, 0.43327394127845764, -0.11503922939300537, 1.1778444051742554, -2.3554975986480713, -0.10305216908454895, 0.18495523929595947, 0.9855908155441284, 1.685145616531372, -0.9959137439727783, -1.2095335721969604, -0.018006427213549614, -0.004058086313307285, -1.2876129150390625, -0.540837287902832, -0.6412445306777954, -0.8843070268630981, 0.24196414649486542, 1.2759761810302734, -0.8160830736160278, -0.27336421608924866, -1.068333625793457, -0.18144913017749786, 0.8195898532867432, 0.6750267744064331, 0.6628641486167908, 0.37828198075294495, 0.6055377721786499, -0.25257351994514465, -2.0422542095184326, 0.3210040330886841, 0.8336607217788696, -0.2809606194496155, -0.4729767143726349, 0.9740802049636841, 1.2038397789001465, -0.413094699382782, -0.0760595053434372, -0.3503616452217102, 0.3787578344345093, -1.2885817289352417, -1.324530005455017, -0.10723250359296799, -0.29401302337646484, 0.4750845432281494, 1.3688349723815918, -0.5440806150436401, 0.979855477809906, -1.1913057565689087, -0.26976943016052246, 0.332564115524292, -0.020364560186862946, -0.7629808187484741, -0.9097224473953247, -0.38372939825057983, -0.8810132741928101, 0.20018228888511658, 0.6130644679069519, -0.4281112849712372, 1.350401520729065, -1.349452018737793, -0.35117512941360474, -0.1076846718788147, 0.2357024848461151, -0.8043636083602905, 1.9082437753677368, 0.31264790892601013, -0.7174656987190247, -0.3009724020957947, 1.0912435054779053, 0.13106048107147217, -1.6863312721252441, -0.7070911526679993, 1.2743335962295532, -0.6863323450088501, 1.075183391571045, -0.44623079895973206, 0.49035540223121643, 1.1729629039764404, -0.7259550094604492, 0.653418242931366, 1.7675715684890747, -0.011895732954144478, -0.06893939524888992, 0.4913018047809601, -1.2417365312576294, 1.1363434791564941, 0.8409295678138733, -0.9859721064567566, -0.003408046206459403, -1.0308773517608643, 0.030626345425844193, -0.7658119797706604, 0.19919100403785706, 0.37996989488601685, -0.6866282820701599, 0.639325737953186, 0.9654354453086853, -0.3545372188091278, -0.36675623059272766, -0.47792357206344604, -0.11450134962797165, -0.9162256717681885, -0.2718874514102936, 1.3776512145996094, -1.3356801271438599], [1.2241404056549072, -1.8912039995193481, 0.45462915301322937, 1.0093178749084473, -1.7310478687286377, -0.4724183678627014, -0.9435540437698364, 1.2080124616622925, 0.36962172389030457, 0.11003462970256805, -0.5256693363189697, 1.4095160961151123, -0.3966018557548523, 1.8690193891525269, 0.6514633297920227, 0.36187416315078735, 1.5022832155227661, 0.11697190254926682, 0.745876669883728, 1.1500166654586792, -0.12574052810668945, 0.7222241759300232, -0.22794407606124878, -0.4677005410194397, -0.5601683855056763, -0.7404592633247375, 0.28669580817222595, -0.5087374448776245, 1.2763748168945312, -0.6274053454399109, -0.7441725730895996, 1.3587560653686523, 0.5073776245117188, -0.22517485916614532, 1.6758557558059692, 0.07278487831354141, 1.1033443212509155, -2.254615068435669, -0.2716893255710602, 0.7774108648300171, 0.09824423491954803, -2.002750873565674, 1.6961878538131714, 0.4816959500312805, -3.0570783615112305, 0.004628901835530996, -0.7958383560180664, 0.10913950204849243, 0.01824866235256195, -0.11255249381065369, 0.09164618700742722, 0.05567308887839317, 0.5439106822013855, 0.38638216257095337, 0.046371132135391235, 1.2182984352111816, 0.08802275359630585, 0.7846323251724243, -2.155360698699951, -0.9481741189956665, -1.7971653938293457, -1.1086161136627197, -0.6705813407897949, 1.7606325149536133, -1.3097585439682007, -1.6988983154296875, -0.41233307123184204, -0.07599753886461258, -1.0856359004974365, 1.071120262145996, 0.6911579370498657, -0.23725664615631104, -0.3007441461086273, -0.531755805015564, 1.5091466903686523, -0.13355636596679688, -0.8905377984046936, -1.5974814891815186, -0.9380766153335571, 0.6218689680099487, 1.9951491355895996, -1.0967484712600708, 1.5103330612182617, -0.3884352445602417, 0.352764368057251, 0.27483096718788147, -0.3623133599758148, -0.17174123227596283, -0.25161781907081604, -0.40221020579338074, 0.5598108172416687, 0.7868035435676575, -0.7661118507385254, 1.055421233177185, -1.2687615156173706, 1.5411994457244873, -1.8437448740005493, 0.6058506369590759, -0.25444814562797546, -0.9464739561080933, -0.3299122154712677, -0.7753753066062927, 1.214371681213379, 0.222465381026268, 0.11471863836050034, 0.14838963747024536, 0.10139085352420807, 0.3329317569732666, 0.10623931139707565, -0.11048226803541183, 0.4858100414276123, 0.776822566986084, -0.6462499499320984, -0.9262245893478394, -0.0028703443240374327, -0.4364677369594574, 0.5215503573417664, -0.7669535279273987, 1.28020179271698, 0.047394759953022, 0.6360417008399963, -0.36210915446281433, 1.5372793674468994, 0.7824316620826721, -0.2135346531867981, 0.8811346888542175, 1.1359162330627441, -0.33610275387763977], [1.3378342390060425, -0.5948139429092407, -0.5915722846984863, -0.2592626214027405, -0.6743910908699036, 0.43278685212135315, -1.359642744064331, -0.1412837952375412, -1.2539488077163696, -1.3224129676818848, 0.20544035732746124, 0.25469449162483215, 0.4510902762413025, 0.7280804514884949, 0.4895568788051605, 1.3908307552337646, 0.1912657618522644, 1.9122538566589355, 0.25211167335510254, 1.0620206594467163, -0.7494754195213318, -1.4218260049819946, 0.9895806312561035, -2.50982928276062, -0.14702782034873962, -2.134840726852417, -1.0843292474746704, 1.8932857513427734, 0.940711259841919, 0.9163946509361267, 0.05839259922504425, 1.5712028741836548, -0.7422766089439392, -0.5218188166618347, -0.8487616777420044, 1.3127681016921997, -0.2276056855916977, 0.47532960772514343, 0.5484076738357544, -1.4007456302642822, -0.8509413599967957, 0.7160811424255371, 0.4690631628036499, -1.7639260292053223, 0.8646480441093445, -0.2819570004940033, 1.0403023958206177, -0.5783858895301819, 0.6698489189147949, 0.727556586265564, -0.5173478126525879, -1.504782795906067, -0.007154344115406275, 1.3917107582092285, 0.5091429352760315, -1.8457242250442505, -0.005377278197556734, -1.7489218711853027, 1.9977704286575317, -0.5302047729492188, -0.5568693280220032, 0.9247210621833801, 3.205634355545044, 1.4479566812515259, -1.764638900756836, -0.017365243285894394, 0.9531481862068176, -0.25519177317619324, -1.023411512374878, -0.0018394084181636572, 0.03420230373740196, -0.386910080909729, 0.33343955874443054, -0.4335557818412781, -1.018092155456543, 2.2339518070220947, 0.3911764919757843, 0.0514385849237442, -1.1688307523727417, -2.011138677597046, 0.21836578845977783, 0.05462266132235527, -0.9603118896484375, -1.4464147090911865, -0.7129002809524536, -0.5499891638755798, -1.7000477313995361, 0.5915897488594055, 1.2983014583587646, 2.091334819793701, -0.29606494307518005, 1.2123034000396729, 0.5378930568695068, -0.18918326497077942, -0.5439935326576233, -0.4431336522102356, -0.4128442704677582, 1.2557207345962524, -0.2408396303653717, -0.13612127304077148, -0.8022617101669312, 0.544037401676178, 0.5610436201095581, -0.9352951049804688, -0.018617814406752586, -0.13497455418109894, 1.4422067403793335, -0.7630881667137146, -0.9534127712249756, 0.04858783632516861, -0.904496431350708, -1.0241777896881104, -0.21917220950126648, -1.5796769857406616, -0.11037302017211914, -0.41879045963287354, 0.5098874568939209, 0.29968345165252686, -1.0936870574951172, 0.5949953198432922, 0.8645495772361755, -1.0078884363174438, -0.021013403311371803, 0.5068768858909607, -1.5602220296859741, -1.347495675086975, -1.1759746074676514, -0.0594957135617733], [0.4336220920085907, 0.9281915426254272, -0.2664167881011963, -0.17262302339076996, -1.3258287906646729, -0.12530776858329773, 0.7684608101844788, 0.753633975982666, 0.5656029582023621, 0.0790737122297287, 0.7428738474845886, 0.07010071724653244, 0.11777576059103012, -0.5483303070068359, -0.26341482996940613, -0.20631077885627747, 1.1889923810958862, -1.4292261600494385, 1.009339690208435, -0.36465510725975037, -1.16751229763031, 0.12911731004714966, -0.41870173811912537, 1.3860357999801636, 1.0378562211990356, -0.009249324910342693, 0.43214723467826843, -0.06707188487052917, 0.29499882459640503, 1.7415379285812378, -0.9659832715988159, 1.5546927452087402, 0.4852816164493561, -2.7219746112823486, 0.6628908514976501, -0.8422412276268005, -0.8315477967262268, -0.7375119924545288, 0.4350717067718506, -0.19511407613754272, -0.12181124836206436, 0.38939929008483887, -0.30113500356674194, 2.0033199787139893, 0.4135552942752838, 0.3857008218765259, 1.1160435676574707, 0.602352499961853, -0.7816261649131775, -0.012463467195630074, 0.10614024102687836, 0.09444674104452133, 1.4841188192367554, -0.31792527437210083, 0.510553777217865, -0.3017442226409912, -0.7529532313346863, 0.944028377532959, 0.4459248185157776, 0.20815342664718628, 0.29300370812416077, -0.23826263844966888, 0.9332671761512756, -2.247735023498535, -0.04787060245871544, -0.5053187012672424, -0.4045732021331787, 1.3473232984542847, 1.0651096105575562, -1.72188401222229, -0.954647958278656, -0.18318963050842285, 0.28849104046821594, 1.026909351348877, -0.5022885799407959, -0.8367801308631897, 1.0957937240600586, -1.0201369524002075, 0.49130329489707947, 0.21616241335868835, -0.7904298901557922, -1.1068322658538818, 1.2339226007461548, -0.4804990589618683, 0.580994188785553, 0.5362223982810974, -1.900059700012207, -0.04822028800845146, 0.20642900466918945, -1.4124553203582764, -0.7289961576461792, -0.654837965965271, 0.8078179955482483, -0.2678716778755188, 0.08225002139806747, -0.9360141158103943, 0.583747923374176, -0.37501809000968933, -0.42895907163619995, 0.1569618284702301, 1.6238254308700562, 0.007973169907927513, 0.4325343668460846, -0.1541176587343216, 0.00199963990598917, -0.8166934847831726, -0.07324357330799103, 0.0016359823057428002, -0.11225605756044388, 1.1111383438110352, -0.854457676410675, -0.30899977684020996, -0.5364142656326294, 0.05311854928731918, -1.719746470451355, -1.2404509782791138, 1.2388489246368408, -0.8542008996009827, -2.1031086444854736, 0.6339455246925354, 0.7002797722816467, -1.8505299091339111, 1.1224768161773682, -1.7151800394058228, -0.3376382291316986, 0.9002172946929932, -0.6181249022483826, 1.7106338739395142], [0.32882729172706604, -0.667702317237854, -0.2209290862083435, -1.1075891256332397, 1.5652259588241577, 0.226410374045372, -0.4143182933330536, -0.2859533429145813, -1.4166022539138794, 0.30886396765708923, -0.4301835894584656, 0.8031443953514099, 1.1475132703781128, -1.834289312362671, 1.7645268440246582, -0.6433315873146057, -0.42341315746307373, 0.8838360905647278, 0.3379875421524048, 1.1270405054092407, 0.5778299570083618, -0.2781842350959778, -0.39207395911216736, 1.1524938344955444, -0.22062058746814728, -0.19275349378585815, -1.5410972833633423, 0.2677605450153351, -0.4670676589012146, -0.8662974238395691, 0.05099334567785263, 0.48555538058280945, -2.179469585418701, -1.5138022899627686, -1.3233639001846313, 0.5258510112762451, -0.23142437636852264, -0.3737109303474426, 0.48176494240760803, 1.8572218418121338, -0.6492233276367188, -1.0905085802078247, -2.383974075317383, -0.6352823376655579, -0.591107964515686, 2.614041328430176, 1.3917968273162842, 2.6060028076171875, -0.5608395934104919, -0.18086330592632294, -0.02025112695991993, 0.856483519077301, 0.5812848210334778, -0.05200723186135292, -1.0195224285125732, 0.013568163849413395, 1.0837664604187012, 0.21926520764827728, 1.268619418144226, -0.9138543605804443, -0.16053104400634766, -0.9747608304023743, -0.18817421793937683, 0.5663440227508545, 0.4580170214176178, 0.11735894531011581, -0.1569480001926422, 0.033350300043821335, 0.05678661912679672, 0.15239977836608887, -0.6562938094139099, 0.43771758675575256, 0.3321959674358368, -0.17627111077308655, -1.2168478965759277, 1.4475113153457642, -1.0537985563278198, -1.4675819873809814, -0.13217154145240784, -0.11725281924009323, 0.40318718552589417, 1.081702470779419, 0.36504316329956055, 0.8504570722579956, 0.9723969101905823, 0.4681374132633209, -0.7277021408081055, 0.4236854314804077, 2.5350260734558105, 1.0426121950149536, 0.7085831761360168, 0.6911405920982361, -2.3435401916503906, 1.7002145051956177, -0.07263597100973129, -0.005948632024228573, -0.8302191495895386, -0.21691206097602844, 0.0011633278336375952, -0.5874432325363159, -0.23721033334732056, -0.36314713954925537, -0.1439327448606491, 2.56801438331604, -1.5241166353225708, -0.6569170951843262, -0.08807235211133957, 0.7264695167541504, 0.23031382262706757, 2.0515544414520264, -0.07244091480970383, -0.7340713143348694, 1.1188242435455322, 0.47233158349990845, -1.3172576427459717, -1.3040999174118042, 0.07283180952072144, 0.39575526118278503, 1.1475989818572998, -1.0279771089553833, -0.6792115569114685, 0.3640610873699188, 0.33529070019721985, 2.205442428588867, 0.05704190954566002, -2.0836310386657715, -0.3049476146697998, -0.6747794151306152], [0.7987890243530273, -1.2366254329681396, 0.9435470700263977, -0.14049537479877472, -0.9767294526100159, 0.24857357144355774, -0.2720690965652466, 0.6720621585845947, -1.0904324054718018, -0.5279877185821533, -1.6516133546829224, -0.30881810188293457, 1.2479852437973022, -1.4370633363723755, 0.051246728748083115, -0.11199092864990234, -1.9328190088272095, 0.6093122959136963, 0.9521611928939819, 1.7119336128234863, -0.3468857705593109, 1.9525060653686523, 0.19912028312683105, 1.0361051559448242, -1.0888721942901611, -0.7226340174674988, -0.4002666771411896, -0.596347451210022, -0.9506891369819641, -0.18219132721424103, -0.3381389081478119, 1.308842420578003, -0.5746435523033142, -0.15270036458969116, -0.4956466257572174, 0.1507035791873932, 0.48895394802093506, -1.5443346500396729, 0.11994273960590363, 0.8259293437004089, -0.3508525788784027, -0.5819219350814819, 1.2442378997802734, 0.42385515570640564, -0.9561456441879272, -0.3007383346557617, -0.13722501695156097, -0.31360822916030884, 1.7263604402542114, -1.6989219188690186, 1.492122769355774, 0.8629629611968994, 0.1428392380475998, 1.4111460447311401, 0.8102352619171143, -0.48123833537101746, -0.5590688586235046, 0.17668646574020386, -1.1808104515075684, 0.9172128438949585, 0.29136574268341064, -0.4470551609992981, -0.7740840911865234, 0.8028327226638794, 0.46032288670539856, 0.7930558323860168, -0.8579411506652832, -2.291823625564575, 0.14565803110599518, -1.3023037910461426, -1.2624377012252808, 0.32755330204963684, 0.3155127763748169, 0.3710591793060303, -0.4032432734966278, -1.6888092756271362, -0.09277143329381943, -0.19900552928447723, -0.8759083151817322, -0.7162295579910278, 0.11471784114837646, -0.471659779548645, -1.9788967370986938, -0.1606580764055252, -0.9815505146980286, -1.4495227336883545, -0.6011474132537842, -0.31312036514282227, -0.665554404258728, 0.13408398628234863, -0.1941346377134323, -1.3494433164596558, -0.6085441708564758, 0.8368613123893738, 1.0984818935394287, 0.2967449426651001, -0.44610127806663513, 2.0720603466033936, 1.124772310256958, -1.023023009300232, -0.2344111204147339, 1.553634524345398, -1.6966298818588257, 0.13261622190475464, -2.7183823585510254, 0.11594255268573761, -1.7742520570755005, 0.8788000345230103, 0.5874079465866089, -0.02105703391134739, -1.5440788269042969, -0.4486396908760071, 1.8580708503723145, -0.595194935798645, 0.8772872090339661, 2.220888614654541, 0.6802460551261902, -0.18302008509635925, 0.24517014622688293, 0.6709724068641663, -0.09853169322013855, -0.8837680220603943, 0.5827907919883728, -0.1880761981010437, 1.0751042366027832, -0.4406180679798126, -1.0740573406219482, -0.22690382599830627], [-0.29758918285369873, -0.24908196926116943, 0.1255388706922531, 1.0285685062408447, -0.7138124704360962, -0.1809324324131012, -0.6634824275970459, 0.3896155059337616, 1.6256132125854492, 1.5060577392578125, -0.22144709527492523, 0.0731920376420021, 1.231227159500122, -0.4713429808616638, -0.5727706551551819, 0.4203380346298218, -0.8290616273880005, 0.2003336399793625, -0.5766350626945496, -0.38420867919921875, -0.13863469660282135, 1.447605848312378, 0.33648326992988586, 0.5023689270019531, -0.31249988079071045, 0.1922333985567093, 0.03938665986061096, 0.7724154591560364, 1.4205228090286255, 1.9357964992523193, 0.20513445138931274, -0.28012681007385254, -0.7294665575027466, 0.03156386315822601, 0.07860836386680603, -0.4140363037586212, -0.27010491490364075, 1.3360390663146973, -0.4569712281227112, -0.28798195719718933, -0.23621536791324615, -0.9422891139984131, 0.2758001685142517, 1.3192075490951538, -0.4111347496509552, -1.7197844982147217, -0.13419297337532043, -0.1721723973751068, -0.047333721071481705, -0.450188010931015, 0.3025374412536621, 1.2682034969329834, -0.5790850520133972, -1.5442291498184204, -0.6166799068450928, 0.14841026067733765, -0.09134821593761444, -1.278326392173767, 0.3812600374221802, 0.29574352502822876, -1.3101222515106201, -2.03802490234375, 0.5042021870613098, 0.835719883441925, -1.4578217267990112, -3.3527369499206543, -0.4282015562057495, -1.0004775524139404, -0.17646220326423645, 0.21198134124279022, 1.0970996618270874, -2.2320058345794678, -0.47377923130989075, 0.7567381262779236, 0.1998952329158783, 0.07551125437021255, 0.33026400208473206, 0.451781690120697, -0.485123872756958, 1.4943077564239502, -1.4414684772491455, 0.256441593170166, 0.001465266803279519, -1.3871550559997559, 1.1892822980880737, -0.4996853768825531, 0.10514821857213974, 0.8644487857818604, -1.429473876953125, 0.14724434912204742, -0.6892127990722656, -1.4226915836334229, 1.323216438293457, -1.6901366710662842, 2.2869865894317627, -0.8712579011917114, 1.3353419303894043, 1.7580716609954834, 0.05030548572540283, -1.03231680393219, -0.5272595882415771, 0.06739204376935959, -0.02167290262877941, 2.1312479972839355, 0.8239810466766357, 0.3635461628437042, 0.5285793542861938, -1.516002893447876, -0.5639108419418335, -1.2300788164138794, 2.117487907409668, 1.140549659729004, 0.4548458456993103, -0.25790560245513916, -0.8744152784347534, 0.8382062911987305, 2.372514009475708, -1.3860340118408203, 0.25147557258605957, 0.6954992413520813, 2.090484380722046, -0.2393483817577362, 0.4957994222640991, -0.5023737549781799, -1.3162544965744019, -1.4619026184082031, -1.0619133710861206, -1.0779552459716797], [1.5074344873428345, 0.9637098908424377, 1.1615523099899292, 0.9841399192810059, 0.7531144618988037, 1.3799242973327637, -0.5776692628860474, 1.2044488191604614, 0.4328003525733948, 0.591914176940918, -0.5680309534072876, -2.2388205528259277, -0.40420717000961304, -0.5870338082313538, -1.1976344585418701, -0.7293894290924072, -0.38823118805885315, 0.44101864099502563, -0.6797179579734802, 0.7308254241943359, 0.18696527183055878, 0.3411244750022888, -1.0608348846435547, 1.8503445386886597, -1.409623146057129, -0.5313829779624939, -0.6462331414222717, -0.2401338368654251, 1.2652653455734253, 0.321087121963501, -0.2871811091899872, -1.3309015035629272, -1.1927629709243774, -1.37529456615448, -1.123026967048645, 0.1783263236284256, 0.047260582447052, 1.9050129652023315, 0.4067777693271637, -2.214811086654663, 2.388899803161621, -1.2117860317230225, 0.16261883080005646, -0.3154792785644531, 0.17861154675483704, 0.14718414843082428, -0.18734285235404968, -2.879868984222412, 0.6642181873321533, 0.6060317158699036, 0.6297414898872375, -0.09720651805400848, 0.30304571986198425, 1.4852652549743652, -1.7153000831604004, 1.1436045169830322, 0.7487451434135437, 1.6506404876708984, 0.6598692536354065, -1.281205654144287, -1.2430609464645386, 0.037152525037527084, 0.862704336643219, 1.4899940490722656, -0.6053754687309265, 0.22252698242664337, -0.04407130554318428, -0.3372665047645569, -0.8839786052703857, 1.0347683429718018, 0.49857062101364136, 0.1671091616153717, -1.735774040222168, -0.020422672852873802, 1.3378897905349731, -1.3262712955474854, 0.7463903427124023, 0.9773009419441223, 0.6985852718353271, -1.2126425504684448, 1.148572325706482, -0.42508649826049805, -0.7894806265830994, 0.7127490043640137, 0.06557086855173111, -0.18666653335094452, 0.21795743703842163, 0.07357543706893921, -0.7075024843215942, -0.6606361269950867, 0.561842143535614, -0.3186090886592865, 0.08627114444971085, 1.2720003128051758, -0.14068855345249176, 0.09518054127693176, 0.024464938789606094, 1.3481459617614746, 0.45813488960266113, -0.3515528738498688, 0.09830482304096222, -1.0204371213912964, 1.3564866781234741, -0.675686776638031, 0.0316695012152195, 0.9488496780395508, 0.25728166103363037, 1.0592230558395386, -0.9675542712211609, 1.4586957693099976, 0.2583216428756714, 0.8921554088592529, -0.1478259563446045, 0.4114735722541809, -1.1503081321716309, 0.13049742579460144, 0.2848053276538849, -1.382531762123108, -0.8389102816581726, 0.8904972672462463, 0.1752949208021164, -0.41388431191444397, -1.403586983680725, -0.09468881785869598, 1.0006532669067383, -0.7614216208457947, -1.9817274808883667, 0.27639949321746826], [-0.5032358765602112, -0.003340542782098055, -0.11965560168027878, 1.3126015663146973, 0.18215055763721466, 0.679131805896759, -0.9096927642822266, 0.8018010854721069, 0.8238006234169006, 0.12344224005937576, -0.1174512729048729, -0.8272635340690613, 0.18648189306259155, -1.5745620727539062, 0.8273571729660034, 0.31797656416893005, -1.5105247497558594, -1.3000644445419312, 0.36106860637664795, 1.2402368783950806, -0.6033441424369812, -0.19400984048843384, 1.3614016771316528, 0.32328277826309204, 0.8847232460975647, 1.1565499305725098, 0.9865489602088928, 0.6984370946884155, 0.1847192496061325, 0.7793717384338379, -1.1327747106552124, 0.6126016974449158, 0.5410255193710327, -1.3088480234146118, 0.663786768913269, 0.03611014783382416, 0.7738773226737976, -0.9060735106468201, -0.7356935739517212, 0.16899770498275757, -1.5676246881484985, 0.7486304044723511, 1.2318861484527588, 0.3706088662147522, 0.3517521023750305, 0.06720516085624695, -0.23969048261642456, -0.5254414677619934, 0.233859583735466, 0.5688246488571167, 0.2030632644891739, -0.843828558921814, -1.6899629831314087, 0.02369229681789875, 0.564449667930603, 0.03876909241080284, 1.4727044105529785, 0.7708796858787537, 0.5390576124191284, 0.8014206886291504, 1.6041570901870728, -0.40729567408561707, 0.6506831049919128, -0.34751248359680176, 1.3731300830841064, -0.886401355266571, -1.0496562719345093, -1.1488077640533447, 0.8654995560646057, -0.27952516078948975, 1.4876741170883179, 0.5089319348335266, -0.2850790023803711, -1.3270350694656372, 0.33518168330192566, 1.1148366928100586, 1.0847196578979492, -0.010544117540121078, 0.7120406031608582, 0.5682191252708435, -0.619132399559021, -0.09077750146389008, 0.46294912695884705, -0.7770837545394897, -0.4377291202545166, 1.6317667961120605, 0.03584270179271698, 1.3287708759307861, 1.2273313999176025, -0.8489993810653687, -1.3675150871276855, 1.3166204690933228, -1.4299019575119019, -1.4104450941085815, 0.7047451734542847, -1.2290605306625366, -0.9694799780845642, 0.7793338298797607, 0.26986023783683777, -0.1532975435256958, -0.1307704895734787, -1.0255608558654785, -0.14339183270931244, -0.2374786138534546, 0.4988630414009094, 0.9278045892715454, 0.494546115398407, -0.7463841438293457, 0.010596255771815777, 0.24651974439620972, 1.1814688444137573, 0.8623521327972412, -0.9940338730812073, 1.2055894136428833, -0.1580866426229477, -1.0765193700790405, -0.012067550793290138, 0.9765331745147705, -2.9146463871002197, -0.8406585454940796, 0.012494250200688839, -0.6719509363174438, 0.9452256560325623, 0.5769603848457336, -0.3149605095386505, 0.2969036400318146, -0.610302746295929, 0.29216840863227844], [1.2855288982391357, 0.002841803478077054, 1.006639838218689, -0.027884354814887047, -0.5094375610351562, -0.11960352957248688, -0.03705023229122162, -0.4643203914165497, 0.5129167437553406, 0.5403375625610352, -0.1542094349861145, 0.4127538502216339, -0.5259762406349182, -0.616432785987854, 0.016894156113266945, -0.7567659616470337, -0.7618122696876526, 0.086773581802845, -0.35171571373939514, 1.1431727409362793, -0.36619192361831665, 0.2190156877040863, -1.5099585056304932, -0.2978112995624542, 1.0493378639221191, -0.7149679660797119, -1.1794606447219849, -1.217300295829773, 0.009880373254418373, 0.7615016102790833, -0.4219297170639038, -1.0627349615097046, -0.12599870562553406, -0.3886890411376953, -1.1930499076843262, -0.3948357403278351, 0.8116588592529297, -0.32413461804389954, 1.3569004535675049, -1.574252963066101, 0.942989706993103, 0.7138884663581848, 0.02634800598025322, -0.11880853772163391, -0.15159671008586884, -1.6668663024902344, 1.6766334772109985, -0.7126299738883972, -0.4260602295398712, -1.467305064201355, 0.35618552565574646, 0.33887460827827454, -0.5290333032608032, -1.9054182767868042, -0.18761491775512695, -0.46853992342948914, 0.2757958173751831, -1.023362398147583, -0.7863461971282959, -1.5999886989593506, -0.018589407205581665, 1.1633599996566772, -0.40457457304000854, 0.9007259607315063, 0.512104868888855, -0.5512697696685791, -0.11986669152975082, -0.7283446192741394, -0.07706040143966675, -2.2403266429901123, 1.263556718826294, -0.47760191559791565, 1.6906813383102417, 1.143147349357605, -0.6216199398040771, -0.0807330310344696, -1.7866793870925903, 0.6086232662200928, 0.9978450536727905, 0.13405737280845642, 1.752837061882019, -0.7816789746284485, 0.7489340901374817, 0.8198813199996948, -0.4639904201030731, 1.0260491371154785, 0.12786531448364258, -0.8687729835510254, 1.3844503164291382, 0.6949285864830017, 0.869166910648346, -0.8319249153137207, 0.08121687173843384, -1.7831534147262573, 0.06924232840538025, -0.2758958637714386, -1.1576308012008667, 0.7776710391044617, -1.972522258758545, 0.12322891503572464, 0.8586305379867554, 0.6843124032020569, -0.15039244294166565, 0.44062793254852295, 0.4549826383590698, -2.9191770553588867, -0.6224925518035889, 1.7535861730575562, -0.3641620874404907, -0.5378440022468567, 0.6748188734054565, 0.00484198983758688, -0.7110225558280945, 0.9061679244041443, -1.5424469709396362, 0.13299603760242462, 0.30674174427986145, -1.1298110485076904, 1.01266610622406, 0.28015202283859253, 0.4651790261268616, 0.2247224897146225, -0.9002721309661865, -0.3949955701828003, -0.25850945711135864, -1.1949869394302368, 0.46336227655410767, 2.087355852127075], [-0.5782905220985413, -0.004968390334397554, 0.8620389699935913, -1.56778883934021, 1.259793996810913, -0.15317197144031525, -0.11753469705581665, -1.0322165489196777, -0.0424027256667614, 0.7999468445777893, -2.0026934146881104, -1.6678804159164429, 1.4145554304122925, -0.5949325561523438, 0.22039711475372314, 0.8751897811889648, -0.8489874005317688, -1.1578532457351685, 0.11523227393627167, -0.7697625160217285, -0.18908870220184326, -0.16064831614494324, 1.908481478691101, 1.6091622114181519, -0.2503412663936615, 0.7748491764068604, -0.13056595623493195, -1.8858458995819092, -0.6795814037322998, 0.813483476638794, -0.5962849855422974, -0.4936445355415344, -0.5996842384338379, -0.6021345853805542, 0.6427406072616577, -0.0923638641834259, 0.2475963681936264, -0.3305060863494873, 1.116005301475525, 0.9809544682502747, 0.9164730906486511, 0.8652864098548889, 1.6293652057647705, 0.9449165463447571, 0.08742530643939972, 0.5311639308929443, -0.9678345322608948, 1.1910113096237183, -0.5599977970123291, -0.569675624370575, 0.10534218698740005, 0.09209167212247849, 1.5749390125274658, -0.4356783330440521, 0.3411317765712738, 1.0174622535705566, 2.32519793510437, -0.5340333580970764, -1.8904272317886353, -2.0087969303131104, 0.17140798270702362, 0.4041251838207245, 1.3266576528549194, 0.11019141972064972, 0.8802706003189087, 1.176878571510315, 0.08808659017086029, 0.651523768901825, 0.5264409780502319, -0.809984564781189, -1.33539879322052, -0.20510199666023254, -0.18148139119148254, -0.26797541975975037, -2.2433252334594727, 1.6520171165466309, -0.06699523329734802, 0.647129476070404, 0.044475264847278595, 0.04005049169063568, 1.4544466733932495, 0.641089677810669, -0.21080338954925537, -0.1081816777586937, -0.5830192565917969, 0.022893907502293587, 0.8449224233627319, -2.1687121391296387, 0.043556395918130875, 0.03035421296954155, -1.6039981842041016, 0.17897705733776093, 0.2578757703304291, 0.0032959168311208487, -0.8858335018157959, 1.050553321838379, 1.2686034440994263, -1.2613893747329712, 1.21591055393219, -0.5165817737579346, 0.7754011154174805, -0.38675156235694885, -1.1419525146484375, -1.0879415273666382, 2.0672149658203125, -1.115775465965271, 0.11193255335092545, -0.6358295679092407, 0.920143187046051, -2.6742045879364014, 0.9014788866043091, 1.34662926197052, 1.6604936122894287, -0.18633145093917847, 1.531242847442627, 0.2297418713569641, 0.8082495927810669, -0.32811233401298523, 0.3815806210041046, -0.023030651733279228, -0.8254783749580383, -0.3740597069263458, -0.07972262054681778, 0.5117880702018738, -0.38648614287376404, -0.14697128534317017, -1.6959433555603027, 1.3176881074905396], [-0.5401821136474609, 0.43391332030296326, 0.9134791493415833, 1.34592604637146, -0.6464133262634277, 1.3680981397628784, 0.5261000394821167, -1.3452180624008179, 0.9716828465461731, 0.7253144383430481, -0.038153816014528275, -1.31004798412323, 0.12763427197933197, 0.7652820944786072, -1.0477758646011353, -0.37539634108543396, -0.8287326693534851, 1.1133776903152466, -0.29609137773513794, 0.07977312803268433, 1.845754623413086, -0.019225554540753365, 1.6698148250579834, 0.4159906506538391, 0.11168563365936279, -0.5794588923454285, -1.3307271003723145, -0.34733590483665466, 0.7145822048187256, -0.0623759962618351, -0.38558775186538696, 1.368929147720337, -0.576217770576477, -0.6884413957595825, 2.4650769233703613, -0.5070674419403076, -1.3075321912765503, 0.5025655031204224, -0.6199051141738892, 0.5260170698165894, 0.041386839002370834, -0.5753629803657532, -1.2985161542892456, 1.1181272268295288, -0.17972561717033386, -0.43484482169151306, 1.8285716772079468, -1.2537990808486938, 1.3244009017944336, -0.2765296697616577, -0.7841223478317261, 0.16040483117103577, -0.9957658052444458, 1.361409068107605, -0.7453945279121399, 0.9718231558799744, -0.5232157111167908, -0.15162020921707153, 0.29211097955703735, -1.508725881576538, -0.08001991361379623, 1.3456370830535889, 1.3025676012039185, -0.3296477794647217, 1.0731914043426514, 0.6248899102210999, 0.4494779109954834, -1.1944142580032349, -0.4461837708950043, 0.5542524456977844, 0.3179912269115448, 0.16951127350330353, 0.1193699836730957, -0.6130433678627014, 0.13494595885276794, -0.5019474029541016, 0.0077836038544774055, -0.12504644691944122, 0.20589132606983185, -0.8693650960922241, 0.8366752862930298, -0.41326239705085754, 0.12116624414920807, 1.6531014442443848, -0.39252060651779175, 0.8892512321472168, -0.14566044509410858, -0.51558518409729, 1.9957199096679688, -0.46510249376296997, 0.41188889741897583, 0.814412534236908, -0.2146858274936676, 1.4031873941421509, -2.3857178688049316, 0.9496837258338928, 0.7029479146003723, 0.7115375399589539, -0.8409765362739563, 0.34346574544906616, -0.3458826243877411, -1.455787181854248, 0.7582194209098816, -1.2821651697158813, 0.015363963320851326, 0.013909822329878807, -1.1492383480072021, -0.8532232642173767, -0.36710718274116516, 0.8199030756950378, 0.6076207160949707, 0.08341756463050842, 0.9296718835830688, 0.4415518045425415, -2.6191251277923584, -1.155652642250061, 0.19979408383369446, 0.15990521013736725, -0.24876363575458527, 0.9710553884506226, -2.2745795249938965, 0.5764601826667786, 0.555464506149292, -0.5121461153030396, 0.9305158853530884, -0.5540280342102051, -0.40703433752059937, 0.8461885452270508], [0.5345526933670044, -1.1290855407714844, 0.4533446431159973, -1.7379117012023926, 0.5390965342521667, 0.8595731258392334, -0.35839641094207764, -0.7231564521789551, 1.820042371749878, 1.3610538244247437, -0.11556629836559296, -0.16306200623512268, 1.0571343898773193, -0.7537211775779724, -0.7193402647972107, -1.5657190084457397, 0.04242367297410965, -1.2884347438812256, 1.5938072204589844, 0.4004996418952942, 1.6133615970611572, -0.5756957530975342, 0.775524377822876, -1.0737656354904175, -0.547045886516571, 1.248000144958496, 1.6123385429382324, 0.7701672315597534, -0.5710331797599792, -0.5253975987434387, -1.1473582983016968, -0.5801324844360352, -1.0933334827423096, 0.510400116443634, 1.986220359802246, -0.007169818971306086, 0.6510542035102844, -0.682456910610199, -0.14631903171539307, -0.9502402544021606, -0.6004624366760254, -1.1658403873443604, -0.10358133167028427, -0.2085476815700531, 0.3680819869041443, -0.8121265172958374, -0.27057430148124695, -0.5779392719268799, -0.7965578436851501, -2.1659250259399414, -0.31210657954216003, 0.31285759806632996, -1.4335825443267822, -0.2498527467250824, -0.7357425093650818, 1.8420445919036865, -0.0016232585767284036, -1.6463509798049927, 0.30928435921669006, 1.3310744762420654, -0.23098976910114288, 1.4588441848754883, 0.930054247379303, 1.0299299955368042, 0.04217320680618286, -0.2561216652393341, -0.9316918849945068, 1.7481136322021484, -1.4088081121444702, -0.545826256275177, 0.5135345458984375, 0.46959611773490906, 0.450784832239151, -0.9169086217880249, -1.6291210651397705, 0.5431144833564758, -1.1829451322555542, -1.103611946105957, 1.4721298217773438, 0.12221391499042511, 1.2517786026000977, -0.8266533017158508, -0.7315818071365356, 0.5297272205352783, 0.8582547903060913, -1.0481164455413818, -1.6356165409088135, 0.1513596475124359, 0.020925795659422874, -0.18246442079544067, -0.7899580597877502, -1.1455998420715332, 2.0530893802642822, -0.4188506007194519, 1.4160083532333374, -1.4521008729934692, -0.1536434292793274, 0.45632338523864746, -1.4270614385604858, -1.4939574003219604, 0.6604012846946716, -0.7497134804725647, -3.0136842727661133, 0.06335270404815674, 0.09688180685043335, -0.12214655429124832, 0.3814624547958374, 0.4065867066383362, -1.7080036401748657, 0.23780126869678497, -0.8438069820404053, -0.6295827031135559, -1.3657681941986084, -0.0043632048182189465, -0.8397501111030579, 0.3008997142314911, -0.6725821495056152, 0.0747278556227684, -0.44838327169418335, 1.4443836212158203, 0.055629223585128784, 0.8320360779762268, -1.5488029718399048, 0.8606275916099548, 1.5083426237106323, 1.2904112339019775, 0.13072772324085236, 1.1580530405044556], [-1.817910075187683, 0.45626962184906006, -0.4247403144836426, 0.24338394403457642, -0.7263693809509277, -0.8628119230270386, 1.8011343479156494, -0.7645900249481201, 1.8726660013198853, -0.3010886013507843, 0.32630786299705505, 2.028127908706665, -0.2773355543613434, 1.110216498374939, 0.4545668661594391, 1.6747914552688599, -0.2982338070869446, -0.20770174264907837, -0.5378998517990112, 0.5022605061531067, 1.359740138053894, 1.375694990158081, 0.225224569439888, -1.4159064292907715, 0.09134388715028763, 0.26405155658721924, -1.6370941400527954, -0.5648118257522583, -0.15448684990406036, -1.29603910446167, 0.02446983940899372, -0.801734209060669, 0.3391760587692261, -0.9832223057746887, 0.06683517247438431, 0.8914731740951538, 1.3349825143814087, -0.3740987777709961, -0.8866808414459229, -2.012333393096924, 0.8641062378883362, -0.7895883917808533, -0.09481481462717056, 0.4534899890422821, -0.4695375859737396, -1.1248294115066528, -2.0377705097198486, 1.4293383359909058, -1.8162819147109985, 0.6818517446517944, -0.23800678551197052, -0.6798643469810486, -0.5511166453361511, 0.32023072242736816, 0.2708474099636078, -0.7690098285675049, -1.097927451133728, -1.996090054512024, -0.33239179849624634, -2.0131328105926514, -1.6669652462005615, 1.710059642791748, -0.11555380374193192, -1.6193861961364746, 1.2329307794570923, -0.5943176746368408, -0.19815035164356232, -0.03099675104022026, -0.3605988025665283, -0.10639891773462296, -0.4976722300052643, -0.24681158363819122, -0.6112176179885864, -0.7256262898445129, -1.6466128826141357, 1.2954704761505127, 1.2093974351882935, 0.8736352324485779, 0.8553639054298401, -0.357724666595459, -0.3551520109176636, 0.9035863876342773, -0.7417911291122437, -0.03382165729999542, -0.3111932575702667, -1.0337399244308472, 0.49196159839630127, -0.3885518014431, -0.20299839973449707, -1.3905786275863647, 0.07513857632875443, -1.3067477941513062, 1.43226957321167, 0.018373413011431694, 0.37151020765304565, -0.17437833547592163, 1.2217477560043335, 2.795158863067627, -1.6076029539108276, -0.8197717070579529, 0.2583700716495514, -0.6355143785476685, 0.2635829746723175, -0.265148401260376, 0.5329124331474304, 1.4266775846481323, 0.07741088420152664, 0.5729529857635498, 0.12629830837249756, -0.36283379793167114, 1.0715279579162598, -0.2443554401397705, -0.7120348215103149, -0.5719737410545349, 0.29483354091644287, 0.7782195210456848, 0.17073313891887665, -1.8443516492843628, 0.34022122621536255, 0.14360526204109192, -0.2064538151025772, -0.05201712250709534, -1.1602991819381714, 0.42513537406921387, 0.4455818831920624, 0.34049326181411743, -1.1672002077102661, -1.426328420639038]], [[-0.008734472095966339, 1.3327014446258545, 0.7636106014251709, -0.72172611951828, -0.01232454739511013, -0.6210401058197021, 0.4611772894859314, -0.6744484305381775, 0.5428645014762878, -0.8509124517440796, 0.12496021389961243, -0.012996514327824116, -1.1891217231750488, -0.0404060035943985, 0.9302752614021301, 0.5238288044929504, 0.9969111084938049, 0.8374224305152893, 0.41341885924339294, 0.6195178627967834, -0.6441766023635864, -2.1219849586486816, 0.79168701171875, -0.23699529469013214, 1.0242860317230225, -1.5693995952606201, -0.9211670160293579, -0.35362452268600464, -0.5392382144927979, 0.5879894495010376, -0.1407274305820465, 0.396860271692276, 0.5416373610496521, -0.04508912190794945, -1.7869246006011963, 0.7160383462905884, 0.24941830337047577, -0.39106252789497375, 0.2943130135536194, -1.1436800956726074, 1.250655174255371, -0.36027494072914124, -0.9643168449401855, 1.4667890071868896, -0.3843594193458557, -1.1657770872116089, 0.013195632956922054, 1.8061721324920654, 0.07621876150369644, -0.4879232943058014, 2.572579860687256, -0.5146059989929199, -0.5017732381820679, -0.3616540729999542, -0.11913440376520157, 2.1090292930603027, 0.5436949729919434, -1.3400276899337769, 0.5644622445106506, 1.9265875816345215, -0.6006386876106262, -0.36005014181137085, 0.36553630232810974, 1.0787760019302368, 1.5031284093856812, 0.06266529858112335, 1.4383362531661987, 1.2298377752304077, 1.4836969375610352, 1.4211469888687134, -0.06527309119701385, -0.3625384569168091, 0.17391802370548248, 0.16333161294460297, -0.8734010457992554, 1.799680233001709, -0.34696751832962036, -0.6914756894111633, 0.7288358807563782, -0.3485962748527527, 0.15229497849941254, 1.1261993646621704, -1.398245096206665, -0.6838545203208923, -0.6500096917152405, -0.9244275093078613, 0.3359050750732422, 0.41457316279411316, 0.840599536895752, 0.8072686791419983, 0.5023140907287598, -0.44468459486961365, -0.8884683847427368, -0.38477733731269836, -1.0431853532791138, -0.8313685059547424, -0.08601560443639755, 0.12885038554668427, -0.5208423137664795, -1.678462028503418, -0.07959253340959549, 1.6202950477600098, 0.36210083961486816, 0.08306003361940384, 0.28839337825775146, -0.756573498249054, -0.634467601776123, -1.6017777919769287, 0.9289376735687256, 0.9688115119934082, 0.4077104330062866, 0.8200308084487915, 0.09414219111204147, 1.2285252809524536, -0.3973698616027832, 0.10827232152223587, 0.013519617728888988, 1.7625746726989746, 0.8261007070541382, 0.7560480237007141, 0.15873074531555176, -0.3243078589439392, 0.46194717288017273, 1.15763521194458, -1.4298715591430664, -0.015359125100076199, -1.3598018884658813, 0.07375548034906387], [0.2606581747531891, 0.8414242267608643, 0.31666263937950134, 0.22333458065986633, 0.03233388438820839, -1.0368787050247192, 0.4019075036048889, 1.8432857990264893, -0.542927622795105, 0.6830213665962219, 0.8489882349967957, -1.5586882829666138, -0.6020258069038391, 1.1416980028152466, -0.08318334072828293, -0.372561514377594, 0.1702144593000412, 0.39788132905960083, -0.08217647671699524, -0.85304856300354, 0.206252783536911, 0.3951651155948639, -2.183959484100342, 0.2151390165090561, -0.42467641830444336, -1.4834073781967163, 1.1240121126174927, 0.5945367217063904, -0.6675605177879333, 1.4920765161514282, -0.888914942741394, -1.4764511585235596, 0.9229094386100769, 1.0581650733947754, 0.4091332256793976, 0.6479783654212952, -0.054461654275655746, -0.7145329713821411, 0.9622433185577393, -1.1812925338745117, -1.7381466627120972, 0.35949012637138367, -0.7683721780776978, 0.8175809383392334, -0.1694502830505371, 0.31303101778030396, -0.6494219303131104, -0.72621089220047, -0.6517340540885925, -1.0282940864562988, 0.5598081946372986, -1.3910211324691772, -0.11608559638261795, -0.2760070860385895, -0.6692334413528442, -1.0017914772033691, 1.8710525035858154, -0.8767194747924805, 1.392391562461853, -0.2013889104127884, 1.2614818811416626, 0.06734522432088852, -1.4770607948303223, 1.2479937076568604, 2.136291265487671, -0.9020664691925049, 1.415140986442566, 0.25434720516204834, 0.5379663705825806, -0.10474051535129547, -0.3761751353740692, -0.7002751231193542, 0.14271336793899536, 0.8437663316726685, -1.13504159450531, -0.5161663889884949, -0.4238704741001129, 0.474547415971756, -1.4555808305740356, -1.2436357736587524, -0.6759857535362244, 0.4588454067707062, -0.1518252044916153, -0.4774424135684967, -0.5730368494987488, -0.46677207946777344, -0.5237608551979065, -0.2197660207748413, 2.564821243286133, -2.4918015003204346, 0.46941009163856506, 1.4637387990951538, -1.024289846420288, -0.004826866555958986, -0.5434427261352539, 0.024378882721066475, 0.40022048354148865, 0.6595008969306946, 1.2075531482696533, -0.8533998131752014, 1.5706217288970947, -0.09712318331003189, -0.6230878233909607, -1.7726902961730957, -0.8817793130874634, 0.7378814220428467, -1.3669406175613403, -2.662369966506958, -0.8578222990036011, 0.036790914833545685, 0.3000647723674774, 0.11723192036151886, 0.7522153258323669, 1.3489652872085571, -0.41054847836494446, 0.08010886609554291, 0.3640812337398529, 0.9714763760566711, -0.7710586786270142, 2.1967554092407227, 0.6044061183929443, 0.2784558832645416, 1.0431804656982422, -0.0031165415421128273, 0.5231770277023315, 0.5278766751289368, -1.437320351600647, 0.009078197181224823], [-0.6898974180221558, -0.2599237561225891, 0.5377057194709778, -0.010250476188957691, -0.5610891580581665, -0.6278742551803589, -0.584911048412323, -1.3396592140197754, 0.6441413164138794, 0.5933056473731995, -1.0217877626419067, -0.41887471079826355, 0.7220399975776672, -0.1029280424118042, -1.4406052827835083, 0.4923602044582367, 1.3452962636947632, 0.029089441522955894, 0.7483141422271729, 0.7961235642433167, 1.1452525854110718, 0.7701051831245422, 0.1777380108833313, 0.2737811505794525, -1.1273070573806763, -0.6768650412559509, -1.0738701820373535, -0.621552586555481, 0.4800385534763336, 1.4280345439910889, 0.8315539956092834, 0.49162259697914124, -1.5914454460144043, 0.4764477014541626, -0.2437644898891449, -0.5872409343719482, 2.4238297939300537, 0.3488822281360626, 0.387087345123291, 1.207189679145813, -1.2483075857162476, -0.3326285183429718, -1.1753205060958862, -1.7177330255508423, 0.2059338390827179, 1.4261442422866821, -0.13752123713493347, -0.9183082580566406, -1.4434038400650024, 1.1495513916015625, 1.616360068321228, 1.8502479791641235, 0.34183669090270996, 0.2997530698776245, 0.4102901518344879, -1.3436199426651, 0.609433650970459, 0.011637352406978607, -0.0068673305213451385, -1.1072099208831787, -0.30893489718437195, 0.09499496221542358, -0.8691544532775879, -0.5614049434661865, -1.834309697151184, 1.0262935161590576, -0.9941180944442749, -1.2103230953216553, 0.47254547476768494, -0.07261226326227188, 0.09126096218824387, -0.8749695420265198, 1.3224117755889893, 0.7153043150901794, -2.2000834941864014, 0.21900460124015808, -0.30728912353515625, -0.8359090685844421, 1.365438461303711, -0.13858072459697723, 0.13151997327804565, 0.7573559880256653, 0.10766952484846115, -0.2155742645263672, -0.905927836894989, -1.0484561920166016, 1.152300238609314, -0.5493123531341553, -0.32671934366226196, 0.41379043459892273, 0.2472054809331894, 1.0524009466171265, 0.9130436182022095, 1.0047814846038818, 1.0344452857971191, -1.244024634361267, 3.218618631362915, 0.7518867254257202, -0.565444827079773, 0.3282626271247864, -0.8119485974311829, -1.0366325378417969, -0.9479910135269165, 0.5570701360702515, 0.3767800033092499, -0.7614184021949768, 0.3717263340950012, 0.7243232131004333, -0.7273723483085632, 0.9470093250274658, 0.9505984783172607, 0.35409122705459595, -1.375520944595337, 0.6176354885101318, -0.11711500585079193, -1.9632917642593384, -0.4194100797176361, -0.1421566903591156, -0.34577786922454834, -0.8143128752708435, 0.12127165496349335, 0.9019173383712769, 1.4508556127548218, -0.1612803339958191, -1.1534743309020996, 0.22287674248218536, 1.3329106569290161, -1.6668914556503296], [1.8934754133224487, -0.19645310938358307, -0.1105315163731575, -0.47450897097587585, 1.8490791320800781, 0.9845614433288574, 1.6644858121871948, -0.36195963621139526, -0.7469369769096375, -0.8836742043495178, 0.47436943650245667, 0.49780771136283875, -1.4801254272460938, 2.4291646480560303, -1.1763427257537842, 0.8570027351379395, -0.8436379432678223, 0.44965696334838867, -0.09291200339794159, -1.0657973289489746, 0.5389553308486938, -0.23796845972537994, 1.2700080871582031, -0.34392663836479187, 0.12366127967834473, 1.4477403163909912, 1.8567208051681519, -0.8355449438095093, -0.4957101345062256, 0.4090616703033447, 1.3860769271850586, -0.7237699031829834, 0.8355410099029541, 0.5179035067558289, -2.2798774242401123, 0.7828624248504639, -0.28422611951828003, 0.3582786023616791, -1.4971730709075928, -0.44801196455955505, 1.310096025466919, -0.09028022736310959, -0.785057008266449, 0.13536258041858673, -0.9385042786598206, -1.4055012464523315, -0.3104725778102875, -0.2095673680305481, -1.313050389289856, 1.7527153491973877, -1.1598519086837769, -1.0087610483169556, 0.06418208032846451, -0.19440577924251556, -0.5296162366867065, -0.39432841539382935, 0.6538923382759094, -1.5044108629226685, 0.24526767432689667, 0.7487166523933411, -1.255448818206787, 0.0741446241736412, 0.17615976929664612, -1.4775855541229248, -0.08563821762800217, 0.7493809461593628, -0.012797525152564049, 0.26419946551322937, 0.7007752656936646, 0.1338832825422287, -0.08343549072742462, 0.40686434507369995, -1.5178934335708618, -1.6657721996307373, -0.9310683012008667, 1.134996771812439, 1.453808069229126, 0.7717480063438416, 1.8562415838241577, 1.0018723011016846, -0.21897824108600616, -0.9656882882118225, 0.22093448042869568, 0.22399598360061646, -1.5342803001403809, 0.055404722690582275, -0.18678328394889832, -0.3500637114048004, -1.1303414106369019, 0.5768859386444092, 1.1774979829788208, 0.15401263535022736, 1.3574388027191162, 1.6910598278045654, -2.5511693954467773, 0.4203789532184601, -1.3585631847381592, 3.0337061882019043, 0.5830872058868408, 1.0782301425933838, 0.5061330795288086, 1.3003133535385132, 1.0630208253860474, -0.9454964995384216, -1.0911909341812134, 0.6852348446846008, 2.743861436843872, -0.13070088624954224, -0.5753509998321533, -0.11062157899141312, -0.011471397243440151, -0.020468369126319885, 0.5566033720970154, -2.4886276721954346, 0.711162269115448, -0.46018025279045105, -0.13873428106307983, 0.7443571090698242, -0.396048367023468, -1.639426350593567, -0.5586645603179932, -2.242363691329956, -0.9157297015190125, -0.9958391189575195, 1.349919080734253, -0.33713120222091675, 0.12626758217811584, -2.2725985050201416], [-1.1531015634536743, -0.6416305303573608, 0.9057729840278625, -2.9811198711395264, 0.4788093566894531, 0.045247334986925125, 0.4282758831977844, 0.6111586689949036, -1.9385473728179932, 0.39631426334381104, 0.08752866089344025, -0.848315954208374, -1.0454685688018799, 2.185856342315674, 0.13146968185901642, 1.4433071613311768, 0.9716131091117859, -0.07205548882484436, -1.087441325187683, 0.41629523038864136, 0.6284359693527222, 0.18074657022953033, 0.5464305281639099, 0.23018072545528412, -1.4632973670959473, -1.5321730375289917, 0.9843500256538391, 2.595773220062256, -0.42482128739356995, -0.4999997019767761, 0.7538524270057678, -0.7813687324523926, 0.6431002020835876, -0.6311472058296204, 0.39643365144729614, 1.1835304498672485, -1.1046446561813354, 0.3467981517314911, -0.20774728059768677, -0.797901451587677, -0.6397404670715332, -0.9832401275634766, 1.2595936059951782, -0.44123584032058716, 0.2477577030658722, 0.528614342212677, -1.951812505722046, 1.1215656995773315, -0.09492133557796478, -0.8255552053451538, 0.8651641607284546, 2.3603551387786865, 0.607792854309082, 0.30405840277671814, 0.8066943287849426, 2.0513689517974854, -0.619776725769043, 0.7431524991989136, 0.8228927850723267, -0.8235979676246643, 0.3808475136756897, -0.2750759422779083, -1.1560174226760864, -0.6178146600723267, 2.2000269889831543, 0.6905520558357239, -0.8830323815345764, 1.5553183555603027, 0.7509939074516296, -0.2537422478199005, -1.0418405532836914, -0.14702825248241425, -1.168997883796692, -0.6316556930541992, -0.4142380356788635, -0.5514976382255554, -1.8876147270202637, 1.469718337059021, 0.18531787395477295, 1.0779504776000977, -0.025439605116844177, 1.6553795337677002, 0.24920515716075897, -0.5723311901092529, -0.07633151113986969, -1.0264819860458374, 0.020365530624985695, -1.098708987236023, -0.8836859464645386, 0.9814292192459106, -0.6911339163780212, 0.7833153605461121, 1.1471375226974487, -0.6347141265869141, -1.1105259656906128, -0.9515364170074463, 0.22681067883968353, -0.18686999380588531, 1.2726881504058838, -1.277477741241455, -0.7554068565368652, -0.720978319644928, -0.1795884668827057, -0.42691007256507874, -1.4875432252883911, -2.0033345222473145, -0.02502250112593174, -1.1425690650939941, 1.7443668842315674, -0.2006857693195343, -0.8698492646217346, -0.08139725029468536, -0.08245261758565903, 0.09630041569471359, 0.32548245787620544, -1.7752301692962646, -0.7541589140892029, 0.06577272713184357, 2.1439170837402344, -0.805996298789978, -0.4788650572299957, 1.6379059553146362, -0.01814660243690014, -1.3118150234222412, 0.8850181698799133, -0.40492939949035645, -1.6085008382797241, -0.6954967379570007], [1.29238760471344, 1.777314305305481, -1.2966244220733643, -0.4891332983970642, 0.0570342056453228, 0.8210768699645996, -1.236331820487976, 0.9739052653312683, 0.3963155746459961, -0.3817862868309021, 0.7046349048614502, -0.4863349497318268, 0.7615129947662354, -1.0198800563812256, -1.0095120668411255, 0.7639187574386597, -0.5635312795639038, -0.0766671746969223, 1.3580011129379272, 1.699464201927185, 0.8697525262832642, -1.092437982559204, 0.17793908715248108, -0.5995723605155945, 0.7827816605567932, -0.6719598174095154, -0.16640537977218628, -0.30549541115760803, -1.0391414165496826, 2.529097557067871, -0.16084304451942444, -0.9141662120819092, 1.6339390277862549, -0.5515540242195129, -0.30249354243278503, 0.10758636146783829, 0.6284343600273132, 0.6889804005622864, 1.0829449892044067, 0.0604410283267498, -0.6148643493652344, -0.3139069378376007, -1.861104130744934, -0.05090972036123276, 0.6800671219825745, 1.825537919998169, 0.2420239895582199, -0.24265915155410767, 0.07845542579889297, -1.693101167678833, 0.4854469299316406, 0.4595872163772583, -0.08858601748943329, 1.1428639888763428, -1.2989174127578735, 0.3874621093273163, 0.2203986644744873, 0.1803501695394516, 0.08561106026172638, -0.7690732479095459, -0.4177524745464325, -0.19636784493923187, -1.4195038080215454, -0.9197543263435364, 0.12536950409412384, -1.0767014026641846, -1.3118693828582764, -0.660323441028595, 1.7444087266921997, 0.4762945771217346, 0.7949167490005493, -1.3207507133483887, -0.706831693649292, -0.8857674598693848, 1.0975443124771118, -0.7315890789031982, -0.3885708749294281, 0.12340714037418365, 1.4850865602493286, -0.005924754776060581, 0.9259847402572632, -0.2533111274242401, 1.3256675004959106, 0.3604159951210022, -0.7645414471626282, 0.3017559051513672, -0.18914054334163666, -0.36451053619384766, 0.6581145524978638, -0.8695579767227173, -0.2584514617919922, -0.6871788501739502, -1.9555736780166626, -0.9132155179977417, 0.9897205233573914, 0.3825764060020447, -0.362735778093338, 0.7770390510559082, -1.9605461359024048, 0.09192486107349396, -0.258851021528244, 2.992511510848999, 0.8309739828109741, -0.8063534498214722, 1.762874960899353, -1.6106387376785278, -0.12426958233118057, -1.1930456161499023, -0.3290063440799713, -0.33241719007492065, 0.3194139301776886, -2.1704399585723877, 1.0817724466323853, 1.0220122337341309, -0.053571559488773346, 0.7758675813674927, 0.2110559195280075, -0.7628532648086548, 0.799382209777832, 1.4612061977386475, -0.8639193773269653, 0.33946213126182556, -1.3278411626815796, 2.192591905593872, 0.11590206623077393, -2.333645820617676, -0.9907547235488892, 0.6430383920669556], [0.45817631483078003, -1.1343663930892944, 2.960028648376465, -1.8767400979995728, 0.6792322993278503, 0.9102358222007751, 0.12089747935533524, -0.5245459079742432, -2.2927401065826416, -2.876542091369629, -1.29815673828125, -0.009412211365997791, 0.43798011541366577, 0.13663052022457123, -0.8422201871871948, 0.1330462396144867, -0.16000711917877197, 0.3624366521835327, -0.07077796757221222, 0.22027982771396637, 2.4619946479797363, -1.3521441221237183, 1.173092007637024, 1.5090559720993042, -1.1880443096160889, 0.45933231711387634, -1.11392343044281, -0.767874002456665, -0.4195460379123688, -0.5437936186790466, -1.5875599384307861, -1.7043697834014893, 0.46937865018844604, -0.07219071686267853, -0.6711511611938477, -1.0394505262374878, 0.30828288197517395, -0.3107098639011383, 0.37157779932022095, -0.5188207626342773, 1.6104680299758911, 0.10626578330993652, -1.1349692344665527, -0.2828681170940399, -0.8766407370567322, -2.1193289756774902, -1.4174782037734985, 1.3759372234344482, -1.1408392190933228, -0.5711217522621155, 0.888237476348877, -0.5518305897712708, -0.63517826795578, -0.7466048002243042, -0.35272929072380066, 0.3530017137527466, 0.7049298882484436, -0.531787097454071, 0.13467249274253845, -0.5398229360580444, 0.7650309205055237, -0.1313951313495636, -1.568372130393982, 0.013945513404905796, 0.9006273746490479, 1.4475172758102417, -1.814875602722168, 1.148776650428772, 0.5554717779159546, -0.23640082776546478, -2.2869160175323486, -0.5524595975875854, 0.7217609882354736, 0.9551239609718323, -3.067800998687744, 0.1027887612581253, -1.2490838766098022, -0.15880827605724335, -0.5617021322250366, 1.2830721139907837, 0.4872911274433136, 0.31820619106292725, -1.6111043691635132, 0.29469698667526245, -0.17593452334403992, 0.3431309163570404, 0.7248058319091797, 0.878054141998291, 1.5793256759643555, -2.084932327270508, -0.3128654658794403, -0.7157660126686096, -1.6433517932891846, 1.0148664712905884, 1.2619966268539429, 0.4228024184703827, 0.3917112946510315, -0.5582108497619629, -0.5362679958343506, -0.6421589255332947, 1.297332525253296, 0.4166681170463562, 1.072312355041504, 1.4767783880233765, 0.9297153949737549, -0.8665104508399963, -0.5698213577270508, 0.6056374907493591, -0.4905220866203308, 1.066330075263977, -1.3788775205612183, 1.2267414331436157, 2.509199619293213, -0.13151152431964874, 0.4661317467689514, -0.08257406949996948, 0.22806310653686523, -0.3267832100391388, -2.0921356678009033, -0.3047087490558624, -0.17373129725456238, -0.22596034407615662, 0.339225172996521, -0.25196948647499084, 0.24693602323532104, -1.6226608753204346, -1.9675021171569824, -0.22794915735721588], [0.38196393847465515, -4.0293169021606445, 0.2165779322385788, 1.1731115579605103, -0.46504515409469604, -0.7643314003944397, -0.9301482439041138, 0.18508148193359375, -0.49202144145965576, 0.1427929699420929, 0.06505756825208664, -0.08312363177537918, -1.378511667251587, -0.8982268571853638, 0.2652210295200348, -1.3968596458435059, 0.4880077540874481, -0.14110340178012848, 1.6271212100982666, 0.3554014265537262, 0.19520540535449982, -0.21773652732372284, 0.10559049993753433, 1.2218247652053833, 0.344488263130188, 0.9308543801307678, 0.3661198019981384, 0.8991259336471558, 0.10556083917617798, 1.9763129949569702, -0.21207694709300995, 1.0659795999526978, -0.18058764934539795, 0.6737472414970398, 0.013778620399534702, -0.326911985874176, -1.100640058517456, 0.6180883049964905, -1.3894424438476562, 0.5114205479621887, -2.007420778274536, -1.222563624382019, -0.19789248704910278, 0.9931193590164185, 1.0184446573257446, 0.9597187042236328, -0.03793143853545189, 0.6720367670059204, 0.36705437302589417, 0.3544791340827942, 0.6099762320518494, -0.9838517904281616, 1.095320224761963, 1.3782730102539062, 1.9833307266235352, 0.9663995504379272, -1.8877722024917603, 2.3298771381378174, -0.7987380027770996, -0.3407191038131714, 0.07080218195915222, -1.212708592414856, 0.8019533753395081, -0.6701036691665649, -0.29844915866851807, -0.4607078433036804, -0.09307020157575607, -1.2140899896621704, -0.9746103286743164, 0.20212915539741516, 0.34054067730903625, 0.1484445333480835, 0.7681865096092224, -1.715949535369873, -2.7413737773895264, 1.3667826652526855, -0.6727025508880615, 0.01563955470919609, -0.47791817784309387, -0.11290350556373596, -1.1080825328826904, 0.9761542677879333, -1.2900365591049194, 1.8130892515182495, 0.35199856758117676, -1.2989591360092163, 0.061350900679826736, 1.478509545326233, -0.2329508662223816, -0.3879130184650421, -1.3972246646881104, 1.5094759464263916, 1.3917204141616821, 0.7293119430541992, -1.1130625009536743, 0.6683861017227173, 0.8372161388397217, -1.2897440195083618, -0.568244457244873, 0.6749973893165588, 0.2594878375530243, 1.3675248622894287, 0.8103231191635132, 1.0721163749694824, -0.4157263934612274, 0.3674667477607727, 1.3488417863845825, -0.8042240142822266, -0.5164837837219238, 0.5034018754959106, -0.5859534740447998, 0.14122281968593597, -0.45749199390411377, 0.4730214774608612, -0.012264042161405087, -1.4429658651351929, -0.8108083009719849, 1.0198520421981812, -1.4093073606491089, -0.04572487995028496, -1.0388340950012207, -0.49959632754325867, 0.6824061274528503, -0.5278589129447937, -1.1812113523483276, 1.025334358215332, -0.05852193012833595, -0.12495677918195724], [0.21397580206394196, 1.04300057888031, 0.01771712489426136, -0.5766514539718628, -0.9962456226348877, -0.2579846680164337, -2.206463575363159, 1.881101369857788, 0.8155649304389954, 0.11217507719993591, -0.5676318407058716, -0.024248402565717697, -0.4997478723526001, -0.5487316846847534, 0.12937340140342712, 0.525463342666626, -0.42658731341362, -0.17685404419898987, -1.8440309762954712, 0.6206604838371277, -0.7879708409309387, 0.9219846129417419, 0.342409610748291, 0.6172705888748169, 0.5789461135864258, 1.4528980255126953, 0.15073134005069733, -0.19297648966312408, 0.08952798694372177, 0.10053087770938873, 2.115978479385376, 0.8907857537269592, -0.40778911113739014, -1.7003086805343628, -0.045513126999139786, 2.5463790893554688, -1.7290165424346924, 0.6639522314071655, -0.5266297459602356, 0.700416088104248, -0.5085262656211853, -0.5572603344917297, 1.4666918516159058, 1.3126089572906494, 0.47848591208457947, 0.008353953249752522, -0.06342098861932755, -0.28103139996528625, -1.3038564920425415, 1.713202714920044, -1.1665886640548706, -0.640962541103363, 0.19832904636859894, -1.6580510139465332, -0.943386435508728, -1.4146281480789185, -0.07088048756122589, -0.976543128490448, -0.673825204372406, -0.04200133681297302, -0.7657538652420044, -0.818664014339447, -0.11122985929250717, -1.5949535369873047, -0.42783546447753906, -1.2740715742111206, 1.794919490814209, -0.7020302414894104, -1.4909031391143799, 0.1392456740140915, -0.4503861665725708, -0.6637601256370544, -0.4395816922187805, -0.6690291166305542, -0.05549001693725586, -0.03447756916284561, 1.6388204097747803, -2.667541265487671, -0.444854736328125, -0.1643296778202057, 0.4471839666366577, -0.9757073521614075, -0.7291184067726135, 0.258943647146225, 0.40036752820014954, -0.8882929086685181, -1.4492368698120117, -0.9935073852539062, -0.4977549612522125, 0.7120065689086914, -0.4077230393886566, -1.0935362577438354, -1.0185301303863525, -0.31578904390335083, -1.7344989776611328, -0.18519684672355652, -1.0149096250534058, -0.5223851203918457, 0.004391548223793507, 0.8719283938407898, 1.0348347425460815, 0.20328988134860992, -0.012500983662903309, -1.150874137878418, 0.8818439841270447, -0.5024991035461426, -0.8288289308547974, -1.5975147485733032, 0.6913832426071167, -0.012783615849912167, 0.4266323745250702, 0.12819251418113708, -1.4398324489593506, -0.27568626403808594, 0.6697953343391418, -0.5132311582565308, 0.8131757974624634, -0.10684414207935333, -0.2331843078136444, 0.8957593441009521, -0.9669171571731567, 0.15613612532615662, -0.7138321995735168, 0.41778257489204407, -0.9385973215103149, 2.0093696117401123, 1.9808623790740967, -0.29653599858283997], [-0.08182678371667862, -0.48493948578834534, 0.18763086199760437, 0.9294368028640747, -0.9128953814506531, -1.5149942636489868, -0.40488582849502563, 1.139797329902649, -1.461622714996338, 0.6385725736618042, 0.9054732918739319, 1.937037706375122, 0.7854561805725098, -1.0751487016677856, 0.8586612343788147, 0.42941802740097046, 0.46127331256866455, -1.4242466688156128, 0.05260614678263664, -0.07367103546857834, 1.118442416191101, -0.07739292085170746, 0.1571149230003357, 0.45338910818099976, -0.9029029011726379, 1.2067281007766724, 1.3478806018829346, 0.9218006730079651, 1.3926069736480713, 0.5389318466186523, -0.2376932054758072, -0.27339306473731995, -1.5061475038528442, -0.7459867596626282, -0.7702087759971619, 0.9882944226264954, -2.4472641944885254, -0.6490247845649719, -0.9626086950302124, 1.3274579048156738, 0.2501228153705597, 0.348891943693161, 1.1835092306137085, -0.18359456956386566, 0.56098872423172, 0.08150264620780945, -0.4457549750804901, 0.2872127294540405, 0.1296558976173401, -0.29945966601371765, -1.0931063890457153, 0.12992259860038757, 1.1483492851257324, 0.31132155656814575, 0.2510010600090027, 0.39023661613464355, -0.10019201785326004, 0.03291346877813339, -0.17897717654705048, -0.9532710313796997, 0.30656734108924866, -0.4180547297000885, -0.49475717544555664, 1.9833308458328247, 1.5232772827148438, -0.7392292618751526, -0.32534530758857727, -0.411975234746933, 1.260818600654602, -1.0562106370925903, 0.955786943435669, 0.17055027186870575, 0.03448905795812607, -0.2367940992116928, -1.366490125656128, 0.7627788782119751, -0.17449909448623657, -0.23430530726909637, 0.3459465205669403, 0.236142098903656, 1.6138371229171753, 0.43457332253456116, 0.153771311044693, -0.6715971231460571, 1.5619525909423828, 1.785581111907959, -0.4614187479019165, -0.5586066246032715, -0.2603716254234314, 1.8634088039398193, -0.4094236493110657, 0.14373625814914703, 0.44159942865371704, -0.6589901447296143, -0.5816476345062256, -0.3868075907230377, 0.6791015267372131, -2.1948022842407227, -0.08772704750299454, -2.005201578140259, 0.23708394169807434, 1.4337480068206787, -1.2070773839950562, 0.20394378900527954, -1.6032013893127441, -0.06501996517181396, -0.08954913169145584, 1.0012315511703491, 0.5678297877311707, 0.651312530040741, -0.7464489340782166, -0.24807658791542053, -1.7036486864089966, 1.2022966146469116, 2.0256834030151367, 0.15326490998268127, 0.5597053170204163, -1.2384495735168457, -0.6249357461929321, 1.3923242092132568, 0.3475455939769745, -0.5449373126029968, 0.1833351105451584, 0.7000798583030701, -0.2468116581439972, 0.9384763836860657, -0.1567886918783188, -0.2358240932226181], [-1.6930326223373413, -0.2840113341808319, -0.204311802983284, 0.12151971459388733, 1.0122554302215576, 0.5995315909385681, -0.49634894728660583, -1.5234962701797485, 0.5837892293930054, -0.7692293524742126, -0.18300706148147583, 0.9805194139480591, 0.2749481797218323, -1.5761675834655762, -1.3576446771621704, 0.8035827875137329, 0.7132003903388977, -0.1691712737083435, 0.2122107595205307, 0.45753875374794006, 1.2021864652633667, -0.5594030618667603, 0.7338692545890808, 0.9766713380813599, 0.29939067363739014, -1.1096214056015015, -0.2148919552564621, 1.9681860208511353, 0.14583152532577515, -0.215717613697052, -1.181923508644104, 0.734446108341217, -1.3127238750457764, 0.4432714283466339, -1.4468719959259033, 0.5030899047851562, -0.6095300912857056, 1.6152822971343994, 0.3013596534729004, -0.015542804263532162, -2.663435697555542, 0.6225888133049011, -1.2636975049972534, -0.1580103039741516, 0.8236801624298096, -0.8098501563072205, 0.41147857904434204, -1.9101594686508179, -0.11353928595781326, 0.3994866609573364, -0.47381359338760376, 2.5949904918670654, 0.27823200821876526, 0.8364174365997314, -0.05279164761304855, 1.577249526977539, -0.5185239911079407, 0.1904952973127365, 0.4357498288154602, 0.4457126557826996, -0.9824533462524414, -0.15648438036441803, 0.004572154022753239, -1.545030951499939, -0.7410544157028198, 0.6958776712417603, 0.6638062000274658, 0.7746741771697998, -0.13266946375370026, -0.224614217877388, -1.601552128791809, -1.5629469156265259, -0.42149364948272705, 0.5633347630500793, 0.9690154790878296, 0.5573760867118835, -1.4823030233383179, 0.40130117535591125, 2.108576774597168, 0.7465946078300476, -0.2691819667816162, 0.678006112575531, -0.2924456298351288, 0.173580601811409, -0.1819387525320053, -2.8736634254455566, 0.5286046266555786, -0.7835663557052612, -0.28985756635665894, -2.305626630783081, 0.6134675145149231, 1.7726976871490479, -0.5752930641174316, 1.7141833305358887, 0.8182504177093506, 0.9450668096542358, -1.1483879089355469, -1.7532365322113037, -0.1868003010749817, -1.0783473253250122, 0.14589087665081024, 0.7427688241004944, -0.5963737964630127, 0.14147132635116577, -0.17583486437797546, -0.09252360463142395, 0.8155333399772644, -0.9340975880622864, 1.3611005544662476, -0.42384329438209534, -0.24464046955108643, -0.7912949919700623, 0.27631402015686035, 0.6803326606750488, -0.5273322463035583, 0.20007413625717163, -0.436069518327713, -0.9398082494735718, -0.7919589281082153, 0.4143153429031372, 0.3691389858722687, 1.2419790029525757, 0.9201148748397827, 1.403600811958313, -0.3391488194465637, -1.7088685035705566, -0.226814404129982, -1.9015798568725586], [-0.783829927444458, -0.6699491739273071, 0.8902294039726257, -0.07000782340765, 1.0039124488830566, 0.42547065019607544, -1.613690972328186, -0.25104692578315735, 1.6043819189071655, 0.3399212062358856, 0.2497270554304123, 1.201418399810791, -0.6913686990737915, 0.29967960715293884, -0.6048392057418823, -0.1478927582502365, 0.19611191749572754, 0.0073818364180624485, 0.56074458360672, 0.25655922293663025, -0.7606425285339355, -0.35001638531684875, -0.26545196771621704, 0.007217556703835726, 1.283019781112671, -0.25888216495513916, -0.026220357045531273, -0.3657929301261902, -0.07256273180246353, -0.9591560363769531, -0.6249162554740906, -1.6280937194824219, 0.26960086822509766, 2.550044059753418, 0.3703993260860443, -0.42431163787841797, 0.035933323204517365, 0.9421348571777344, 0.26723870635032654, -0.36199432611465454, 0.34942537546157837, 0.08025909960269928, -0.6877235174179077, -0.7615296244621277, -0.13374899327754974, -2.146260976791382, -0.0023066725116223097, 1.0866895914077759, 0.26138070225715637, -0.5555503964424133, 0.8321210741996765, -1.0665584802627563, 0.28760480880737305, 1.2630971670150757, -0.7096045613288879, -0.1639525443315506, 1.0895898342132568, 0.8288533091545105, 0.9956247806549072, -0.18169225752353668, 1.2149320840835571, 0.46040210127830505, -1.6295274496078491, -0.4659058153629303, 0.22634363174438477, 0.3120606243610382, -1.8052634000778198, -0.18333357572555542, -2.4682557582855225, 0.3782191574573517, -0.3905681073665619, 0.09701873362064362, 1.096605658531189, 1.6403899192810059, 1.3634575605392456, 0.374778687953949, 0.8573644161224365, 0.5624415278434753, 0.7976146340370178, 0.7444034218788147, -0.013349964283406734, -0.5129961967468262, -1.7735458612442017, 1.9600476026535034, -0.668979823589325, -0.8030834197998047, 0.14897428452968597, 1.51140296459198, -0.43729254603385925, -1.0908937454223633, -1.513734221458435, -0.09002263844013214, -0.02532055787742138, 0.5434502363204956, 0.518565833568573, -0.041504304856061935, -0.3622383773326874, -1.749922275543213, 1.3582960367202759, -0.5730890035629272, 1.5041308403015137, -0.7768863439559937, -0.2014537751674652, -1.00368070602417, 0.2930629253387451, -1.242531180381775, 0.17302505671977997, -1.9498730897903442, 1.4753258228302002, 0.7323043942451477, 1.4072808027267456, -0.9144154191017151, -0.7547871470451355, -0.9817494750022888, -1.0858886241912842, -0.3440057933330536, 0.7467668056488037, 0.3862607181072235, -0.19044898450374603, -0.4772975444793701, 0.7327473163604736, -0.7412974238395691, -0.7843727469444275, -1.962753415107727, -1.7418278455734253, -1.826945185661316, -0.16399995982646942, -0.6774344444274902], [-0.4934457242488861, 2.5194783210754395, 0.15016524493694305, 1.0531398057937622, -0.847176194190979, 0.3073199987411499, -0.05287324637174606, 1.789304494857788, -0.6815351843833923, -0.356868714094162, -1.8245220184326172, 0.9596942067146301, 0.14084357023239136, -2.117366075515747, -0.5126624703407288, -0.22452344000339508, 0.6660440564155579, 0.9706974029541016, -0.44504499435424805, -0.1451282948255539, -0.10294469445943832, 0.7624244093894958, 0.23471999168395996, -0.4448646605014801, 0.686343789100647, -1.7529300451278687, -0.4675178527832031, 0.7248672246932983, 1.7772905826568604, -0.3604591190814972, 0.40353643894195557, -0.21595747768878937, -0.12859059870243073, 0.12901246547698975, -1.1390595436096191, -2.222428560256958, -0.5634883642196655, 0.28218722343444824, 0.5330188274383545, -0.019471392035484314, -0.13000617921352386, -2.4695911407470703, 0.031096970662474632, 0.762319028377533, -0.9979803562164307, 0.49896758794784546, -0.1676894724369049, 1.5650300979614258, 0.5428798794746399, 0.0030422115232795477, 0.28889691829681396, 0.3733527958393097, -0.45885682106018066, -0.23530572652816772, 0.830369234085083, 0.9506152868270874, -0.2525882124900818, -1.1348947286605835, -2.1659889221191406, 0.7877999544143677, -0.632259726524353, -0.7692251801490784, -1.3674763441085815, 0.6403077244758606, 1.9542746543884277, -1.4765299558639526, 0.9024468064308167, -1.769095540046692, -0.9555574059486389, -0.9986515045166016, -0.8584395051002502, -0.23622702062129974, -0.1990777999162674, -0.9950893521308899, -0.4852046072483063, -1.117372989654541, 0.8495969176292419, 1.6299432516098022, 0.06124820560216904, -0.8714560270309448, -0.3964475095272064, 0.3608298897743225, -0.5641316771507263, 1.0742626190185547, 0.6575286388397217, -0.45259925723075867, 1.018876552581787, 0.2560640275478363, 1.5992811918258667, -0.20558024942874908, 0.9500181078910828, 0.609281599521637, -1.0103083848953247, 0.6065719127655029, 0.281229704618454, -1.3674060106277466, -1.8074798583984375, -1.633097529411316, 0.8913966417312622, -1.2654125690460205, -1.1807953119277954, -0.13014325499534607, -0.5526072382926941, 0.1563921719789505, 1.1820783615112305, 0.1569642573595047, -0.18442794680595398, 1.7105605602264404, 0.39972150325775146, 2.177337884902954, 1.8781567811965942, -0.24984519183635712, -0.29688504338264465, 0.8080618977546692, 1.711159348487854, -0.12743046879768372, -0.4777509272098541, 0.7528270483016968, 0.2440406084060669, 0.6421346068382263, -0.2965739667415619, -1.01736319065094, -1.7214841842651367, -2.2237777709960938, 0.005262542515993118, -0.7248466610908508, -0.01544992532581091, -0.2795834243297577], [-0.6323152780532837, -0.5643066167831421, 2.463148355484009, -0.5675698518753052, -0.46649518609046936, -0.4498867690563202, 0.770984411239624, 0.3545266091823578, 0.11140631139278412, 0.9392080307006836, -0.984076738357544, -0.37558525800704956, 0.587222158908844, 0.18978412449359894, -0.7003217935562134, -0.5840687155723572, 1.710982084274292, 0.278085321187973, -0.04503077641129494, 0.673328161239624, -1.1823248863220215, -0.4309571087360382, 0.29765936732292175, 0.5430475473403931, -0.20932415127754211, 1.2750656604766846, -3.5466325283050537, 0.3240337371826172, 0.8515687584877014, -0.19474905729293823, -0.6169548630714417, -0.39771568775177, -0.4744527041912079, 0.7522315979003906, 0.2804078459739685, 1.5144128799438477, -0.6933291554450989, 0.1400422602891922, -1.5409839153289795, 1.0440176725387573, 0.5602430105209351, -0.061061907559633255, 0.18562380969524384, 0.23935581743717194, -0.45856326818466187, -1.164552092552185, 0.6399939060211182, 1.5912914276123047, -0.6700191497802734, 1.3061347007751465, -0.7404232621192932, -1.3512458801269531, -0.5792381167411804, -0.8665628433227539, -0.23914356529712677, 0.46055641770362854, 0.027379287406802177, 0.025117116048932076, -0.6488712430000305, -1.0294438600540161, 0.6873692870140076, 0.2707534432411194, -0.35619693994522095, -0.8356716632843018, 2.0929503440856934, -1.4114015102386475, 0.1159788966178894, 0.14497266709804535, 1.7151157855987549, 1.0621590614318848, -0.894227147102356, -1.4664674997329712, 1.1108038425445557, -1.3851858377456665, 0.6643754839897156, -1.0109301805496216, 0.17591461539268494, -1.591963768005371, -0.2742287218570709, 1.9880536794662476, 1.57386314868927, -0.11622651666402817, 1.487572193145752, 0.4911497235298157, -1.3000346422195435, -0.6360448598861694, -2.026500701904297, 0.6210840940475464, -0.9902346134185791, -0.9013968110084534, -1.4270437955856323, 2.4291868209838867, -0.4938393533229828, 2.057197332382202, -0.3594268560409546, 2.7899534702301025, -1.209814190864563, 0.5048532485961914, 0.997687578201294, 1.3742612600326538, 1.6614820957183838, -0.5279254913330078, -0.729452908039093, -0.9026844501495361, 0.5289773344993591, -1.3553662300109863, -0.3524569869041443, 0.7775131464004517, -0.4175679087638855, -0.11005880683660507, 1.1535344123840332, 1.0183818340301514, -1.160442590713501, 0.4902423322200775, 0.019692106172442436, 1.0609841346740723, -0.8196881413459778, 0.5484899878501892, -0.4171235263347626, 0.2976877987384796, -0.7642583250999451, 1.6179859638214111, -0.2844104766845703, 0.6231817603111267, -0.9385481476783752, -1.1213418245315552, -0.8258628249168396, -0.602936327457428], [-0.11095781624317169, -0.2571030557155609, -0.4047876298427582, 1.5859694480895996, 0.27882951498031616, -0.005149048753082752, -0.9060094952583313, -0.5257459878921509, -1.2680399417877197, -2.140763998031616, 0.7495003938674927, 1.537514567375183, -0.25917714834213257, -0.17197853326797485, 1.3963528871536255, 0.750033438205719, 0.5712385773658752, -0.006336594466120005, -1.3493537902832031, -0.17362354695796967, -0.6333430409431458, -1.876781702041626, 1.9438809156417847, 1.2166677713394165, -0.18232716619968414, 0.46752429008483887, -0.7634376287460327, -0.4877503514289856, -0.6154467463493347, 1.2189189195632935, -0.29148420691490173, -0.78349769115448, -0.5518575310707092, -0.5887243747711182, 1.2973288297653198, -0.10051143169403076, 2.5366063117980957, 1.1098697185516357, 0.554206132888794, 0.31298258900642395, -0.7778330445289612, 1.14589524269104, 1.0820928812026978, 0.19568109512329102, -0.6609970927238464, -1.543216586112976, 1.4911057949066162, 1.8168913125991821, 1.7013789415359497, 1.33324134349823, -0.5531721711158752, 0.17532648146152496, -0.5757947564125061, -0.7966152429580688, 1.438712477684021, -0.6461631655693054, 0.03541124612092972, -1.0276752710342407, 0.45026835799217224, -0.26045358180999756, 0.4131101965904236, 0.8088595867156982, -0.6750319600105286, 1.7092947959899902, 1.9352790117263794, 1.1248204708099365, -1.6068240404129028, 1.3696262836456299, -1.9313652515411377, 1.1618657112121582, 0.4576318562030792, 0.6309959292411804, -1.7402127981185913, 0.025349527597427368, -1.2979761362075806, -0.4697130620479584, 0.06636709719896317, 1.006467342376709, -0.39791345596313477, 1.84469473361969, -0.6281697750091553, -0.5476555824279785, 0.8660798668861389, 0.5671270489692688, 0.8072747588157654, 0.24637340009212494, -0.8780646920204163, -1.471300721168518, 1.348764419555664, -1.2755134105682373, -0.3957444727420807, -1.1323442459106445, -0.24962107837200165, -0.4493982493877411, -1.5617825984954834, -0.5810452103614807, -1.2916404008865356, 0.43224963545799255, 0.4350689649581909, 0.43570929765701294, 0.636674165725708, 1.032160997390747, 0.3417280912399292, -0.8965650796890259, -0.3087373971939087, -0.23659217357635498, -0.9183416962623596, 0.9085791110992432, -0.8442639708518982, -0.7605013251304626, -0.20355914533138275, -1.401365041732788, -0.9285013675689697, -1.0484983921051025, -0.8434883952140808, -0.8354001641273499, 1.0846093893051147, -0.5360236167907715, 0.13793890178203583, -0.7151302695274353, 0.21877306699752808, -1.2808386087417603, -0.6041355729103088, -2.849501132965088, -0.6736730933189392, -0.8044220209121704, 0.08391494303941727, -0.814704954624176], [-0.8427391052246094, 2.3176846504211426, 1.495679497718811, -0.33588090538978577, -0.7185012102127075, -0.32831889390945435, -0.9954744577407837, -0.9221412539482117, 0.10117282718420029, 0.7456324100494385, 1.6155061721801758, -2.1835720539093018, -0.3078606128692627, -0.5459253191947937, -0.2787478566169739, -0.008335855789482594, 1.2434316873550415, -0.23145005106925964, 0.2687374949455261, -0.2868548333644867, 0.676002025604248, 1.4622119665145874, 0.7165082693099976, 0.7712153792381287, -1.4784960746765137, 1.016833782196045, -0.2621106505393982, 0.10167568176984787, -0.5751744508743286, 1.1896296739578247, -0.23583127558231354, 0.75334233045578, 0.6287490725517273, 0.26634085178375244, 1.1543192863464355, -0.25574469566345215, 1.2370755672454834, 0.640067994594574, -0.4392206370830536, -1.0225257873535156, -0.4246237874031067, -1.976402997970581, -0.5255288481712341, 1.4799861907958984, 1.6161688566207886, -0.3474765121936798, -1.4231830835342407, 0.4031751751899719, -0.19604647159576416, 1.103330373764038, -1.3719301223754883, 1.200135350227356, 1.505537748336792, 0.35877931118011475, -1.1993390321731567, -1.0789142847061157, -0.07638653367757797, -0.5524026155471802, 0.6955615282058716, -0.38423770666122437, 0.9045871496200562, 1.315445899963379, -0.417015939950943, -0.5273891091346741, 0.7849816679954529, -0.04965026676654816, -0.5108321309089661, -0.518927812576294, 0.732189416885376, 0.9954941272735596, -1.0724949836730957, -0.226702019572258, -0.39387622475624084, -0.12011582404375076, 1.1526507139205933, 0.478491872549057, 0.533713161945343, 1.1156007051467896, 0.17297376692295074, 0.6201508641242981, 1.0060410499572754, 0.9711675047874451, 0.4112713634967804, 0.7853886485099792, 2.953662872314453, -0.6371814608573914, 0.4901692569255829, 1.4747916460037231, 0.3842867314815521, 0.02172093093395233, -0.2870345413684845, 1.029006838798523, -0.03476522117853165, -0.9570298194885254, 0.48406219482421875, -0.2162783443927765, 0.14705795049667358, 0.2132178544998169, -1.4254359006881714, -0.8707214593887329, 0.12446340918540955, 2.4044859409332275, 1.5026466846466064, -0.4126266539096832, -1.0589497089385986, 1.3677022457122803, 1.008388638496399, -0.01620817743241787, -1.1621471643447876, -0.4743008315563202, 0.18092823028564453, -1.0779235363006592, -0.9075405597686768, -0.3387173116207123, 0.50214684009552, 0.16481617093086243, 0.2655441462993622, -0.3982042968273163, 1.775809645652771, 0.9060655236244202, -0.0540505088865757, -0.12045391649007797, 0.2533249855041504, -0.7724977135658264, -0.8121533393859863, 0.7651538252830505, 0.15281827747821808, 1.6373132467269897], [0.15053145587444305, 0.9304575324058533, -0.384578675031662, -0.007367467973381281, 1.5291823148727417, 2.006168842315674, 0.650174617767334, -1.0451297760009766, 1.4153655767440796, 0.6655828952789307, 0.4385821223258972, 0.05299137532711029, -0.1473310887813568, -0.994932234287262, -1.4384360313415527, -3.1190075874328613, -0.2252018004655838, -0.4004628658294678, -0.7592173218727112, -0.6159814596176147, 0.4269867539405823, -0.6627020835876465, -0.07285512238740921, 0.9340572357177734, 0.8248418569564819, 0.33478617668151855, -0.24994277954101562, -1.0571311712265015, -0.4502105712890625, -0.5964949727058411, 0.3223753273487091, -0.23082466423511505, 0.9868380427360535, -1.0217615365982056, -0.4417850375175476, 1.0093873739242554, -1.2848501205444336, -1.269415020942688, -1.679500937461853, -1.432144284248352, 0.29687345027923584, -1.2299513816833496, -1.0226808786392212, -1.0762097835540771, 1.4282833337783813, 2.711764335632324, 1.5677131414413452, 0.8031371831893921, -0.3733691871166229, -0.6423697471618652, -0.9447156190872192, 1.472653865814209, -2.153801679611206, -0.6387927532196045, -0.5212746858596802, 0.4784979522228241, -1.6575027704238892, -0.0007630400359630585, -0.7371842265129089, 0.7819066643714905, 0.7884593605995178, 0.3683070242404938, 0.8542937636375427, 1.257912039756775, 0.9563064575195312, -1.5504435300827026, -1.997555136680603, -0.05390524864196777, 2.1054000854492188, 2.4056458473205566, -0.6966675519943237, -1.4320993423461914, -0.611469030380249, 0.5827867388725281, -0.8432411551475525, 0.9465194344520569, 1.2839473485946655, -0.08707716315984726, -0.1953684687614441, 0.8607439994812012, 0.7654643058776855, -0.031261228024959564, 0.3131684958934784, -1.5294498205184937, 2.7019591331481934, 0.11374101787805557, -1.1849746704101562, 0.7781582474708557, -0.6787111759185791, -0.15043699741363525, 1.5669749975204468, 0.14074033498764038, -0.4027341306209564, -0.7281877398490906, -0.3114010691642761, -0.4599674344062805, 0.13568903505802155, -1.2331093549728394, -0.5799238681793213, -0.5165960788726807, 0.7169005870819092, -0.0963490754365921, -0.5402697324752808, -1.2886320352554321, 0.613208532333374, 0.03196243941783905, -1.404940128326416, 0.022539813071489334, -0.2678877115249634, -1.6616209745407104, -0.40382876992225647, 1.5014067888259888, -0.5921662449836731, 1.135225534439087, 0.4648467004299164, -0.6513588428497314, 1.1906065940856934, 0.06873651593923569, 0.1181112751364708, -1.3569207191467285, -0.623245894908905, -1.1401066780090332, 0.6458588242530823, -0.8178001046180725, -1.0215898752212524, 0.6013060212135315, -0.8223710656166077, -1.014479637145996], [0.5087855458259583, -0.3565680980682373, -1.9081956148147583, 0.8798692226409912, -0.8165145516395569, 1.9846818447113037, -0.5113517642021179, -1.280795931816101, 1.0910042524337769, -0.7449092864990234, -0.613734245300293, 1.7602170705795288, 0.3677506446838379, -0.330753356218338, 1.2863212823867798, 0.06994892656803131, 1.0446176528930664, -0.031232908368110657, 0.23147797584533691, -0.8319517970085144, -0.4888160526752472, -0.591232419013977, 0.6883261799812317, 0.38012826442718506, -0.17117169499397278, 1.4480823278427124, 0.2375091314315796, -1.1409820318222046, 0.5632595419883728, 0.729250431060791, 1.143095850944519, -0.010269759222865105, 0.6134427785873413, 1.5999969244003296, -0.38163942098617554, 2.257784366607666, 0.8875465393066406, 2.099238395690918, -0.3947252333164215, 0.17330145835876465, -0.7198073863983154, -0.7760651111602783, 0.20769882202148438, -0.5901899933815002, -1.1631792783737183, -2.345099449157715, 0.022501369938254356, -1.263706922531128, -0.4460015892982483, 0.3963383734226227, -2.2125749588012695, 0.9749841690063477, -0.1340314894914627, 0.010065197944641113, 0.9868162274360657, -0.8809483647346497, 1.595190405845642, 1.1194013357162476, -0.4483692944049835, -0.09358025342226028, 0.6255583763122559, 0.33008989691734314, 0.8078299760818481, -0.6645509004592896, -0.9805921912193298, -0.0875156819820404, 0.23600485920906067, -0.39459046721458435, 1.36549711227417, -0.8959894180297852, 0.8164353966712952, 1.122449517250061, 0.9426388144493103, 0.463095098733902, -0.6103013753890991, -0.7199363112449646, 1.728883147239685, -0.5505810379981995, 1.5259554386138916, -0.6450443267822266, 0.016518739983439445, 0.37398484349250793, 1.8744937181472778, 0.03218473866581917, -1.6206331253051758, 0.5020762085914612, 0.6018590331077576, -0.6684430837631226, 0.587135910987854, -2.6101033687591553, -0.04867630451917648, 0.19602760672569275, -0.8456482291221619, -1.4795578718185425, -1.7429757118225098, -0.5988441109657288, -0.266830176115036, 1.4396296739578247, 0.8331855535507202, -0.786307692527771, -0.13801684975624084, 0.863386332988739, -1.132095217704773, -0.8800965547561646, 0.38612428307533264, 0.8668399453163147, 0.29838263988494873, -0.277545690536499, -0.5635042786598206, 0.48231297731399536, -1.2254939079284668, -0.5445601344108582, -0.4950577914714813, 0.15490348637104034, 0.7924984693527222, -0.1276961863040924, 0.41640704870224, -1.3471680879592896, -0.7452226877212524, 1.057694435119629, -0.7740607857704163, 0.18979467451572418, -0.13707159459590912, -0.11946873366832733, -1.3376425504684448, 0.3308563530445099, 0.5897204875946045, 2.389309883117676], [-0.6865536570549011, 1.265928030014038, 0.8965938091278076, -0.27378711104393005, 2.0328307151794434, 0.45057687163352966, -0.19379335641860962, 0.4260920584201813, 0.4197944402694702, -0.6386586427688599, -0.7893370985984802, -0.2119634747505188, 1.6688580513000488, 0.8754501342773438, 0.4495597779750824, 0.1041162833571434, -0.8585598468780518, 1.1711699962615967, 1.4645293951034546, -0.42763909697532654, 0.05901283398270607, -0.5381608009338379, 0.3664151430130005, 1.2595245838165283, -0.5870397090911865, 0.6677567362785339, -1.5949647426605225, 0.41824957728385925, -1.0936203002929688, 0.2590384781360626, -0.3240397572517395, 0.6085789203643799, -1.600229024887085, -1.1643706560134888, -0.25114139914512634, 0.5891391634941101, 1.5365523099899292, -1.4016557931900024, -2.4180660247802734, -0.3165042996406555, 1.16932213306427, 0.11203933507204056, 0.7875513434410095, -0.8860413432121277, 1.4029700756072998, -2.111210584640503, -0.7715858221054077, -2.8536272048950195, -1.9782274961471558, -0.31646397709846497, 0.3554675579071045, -0.687561571598053, -0.3605266213417053, 0.44870615005493164, -0.3044907748699188, -0.3373083174228668, 0.8277751207351685, -0.5248220562934875, -0.8275430202484131, 0.3517323136329651, -2.2713096141815186, 0.3350382447242737, 0.6370731592178345, -0.2511066198348999, 2.4073898792266846, 0.6001157164573669, -0.9628444314002991, -1.2814624309539795, -1.1547633409500122, -1.090219259262085, -0.37075385451316833, 0.9780303239822388, -0.5176101326942444, 1.8680719137191772, -0.48844820261001587, -1.4046169519424438, -0.5122953057289124, 0.6844784021377563, -1.1313647031784058, 0.14067065715789795, 0.4874141216278076, 0.3183024227619171, -1.4564827680587769, 0.520081102848053, 1.186286211013794, -0.12975040078163147, -0.7730720639228821, -0.167343869805336, 0.39000192284584045, -1.5853745937347412, 0.055042605847120285, 0.3717549741268158, 1.7435506582260132, -0.15543681383132935, 0.294831782579422, 0.40516868233680725, -1.2718595266342163, -0.3966929316520691, 0.004678498022258282, 1.8892425298690796, 1.1282235383987427, 1.7869935035705566, -0.6721200942993164, 1.4276350736618042, 0.1746438890695572, -0.08576696366071701, 0.8381829261779785, 0.43995726108551025, 0.4954344630241394, -1.1133854389190674, 0.38916507363319397, 0.4562324583530426, 0.05105608329176903, -1.5225071907043457, 2.17743182182312, 1.1219562292099, -1.2329668998718262, 0.4204833209514618, 1.1538960933685303, 0.32842662930488586, -1.0390329360961914, -0.6470673084259033, -0.3993928134441376, 1.0280109643936157, -0.32096585631370544, -0.45836231112480164, -0.5461694598197937, 0.29042091965675354], [0.2632085382938385, 0.3070344924926758, -0.9003698229789734, -1.7731274366378784, 0.11366105079650879, -0.5718843936920166, -1.7713145017623901, -1.0228995084762573, -1.0447719097137451, -0.4160110056400299, -0.016727743670344353, -0.646571159362793, 1.5053753852844238, 0.5040534734725952, 0.8720205426216125, -0.05064714327454567, -1.6465190649032593, 0.9769951701164246, 1.6160911321640015, -0.12677690386772156, -1.545426368713379, 0.3444035053253174, 0.7151625752449036, -3.0939931869506836, -1.3619672060012817, 0.49962061643600464, 0.26170119643211365, 0.6799102425575256, -0.7575814127922058, -0.09271416068077087, 0.2640286087989807, -0.13547855615615845, 0.9675052165985107, 1.1900912523269653, 0.6276452541351318, -0.09921993315219879, 0.0326094850897789, -0.22219257056713104, -0.033755674958229065, 0.6758436560630798, 1.3647966384887695, 0.6535900831222534, -0.41275832056999207, 1.0063462257385254, 0.7148202061653137, 1.101693868637085, -2.1417770385742188, 0.9450786113739014, -1.356092929840088, -0.07239269465208054, 0.19374744594097137, -0.9272586703300476, 0.6146535873413086, 1.634464979171753, -1.5651484727859497, 0.7437136173248291, 2.1410040855407715, -0.8091065883636475, -0.39291343092918396, -0.8113666772842407, -0.7948471307754517, -0.8087158203125, -0.4559457004070282, -0.23410166800022125, 0.5700506567955017, -0.12262942641973495, -0.7516908049583435, -0.5522962808609009, -0.38950878381729126, -1.8811286687850952, 1.0989426374435425, -0.4112260043621063, -0.8211973905563354, -0.04744937643408775, -1.1528321504592896, -0.19077923893928528, 0.11491918563842773, -0.1391848623752594, -1.1382471323013306, 0.7225068211555481, -0.19389256834983826, 0.7932407259941101, -1.2604464292526245, -0.587956428527832, -0.36439448595046997, -0.48466384410858154, 0.9707752466201782, -0.33753570914268494, 2.3066279888153076, 2.4312214851379395, 2.2859091758728027, -1.04962956905365, -2.3461835384368896, -1.2965683937072754, -1.3498324155807495, 0.4178873300552368, -1.3834527730941772, 1.6930407285690308, 0.7678627371788025, 0.3402889668941498, -0.16580364108085632, 1.172284483909607, 0.7071788907051086, 1.1637835502624512, 1.5721200704574585, -1.1481094360351562, 0.3560290038585663, -0.9633552432060242, 0.5684093832969666, 0.6283484697341919, 1.2714462280273438, 0.010106203146278858, -0.4633101224899292, -0.18541020154953003, 0.8562650680541992, -1.0852503776550293, -0.6146815419197083, -1.1037981510162354, 0.5177047848701477, 0.4473923444747925, -0.9094889760017395, 0.4672684371471405, 1.8215970993041992, 0.057746510952711105, 0.07614292949438095, 2.184782028198242, -1.1481002569198608, 0.2950226068496704], [0.5394085049629211, -2.3969452381134033, -0.346049040555954, -1.6918160915374756, 0.022302409633994102, -0.3134192228317261, 1.7472913265228271, -0.5467780828475952, -2.0574791431427, 1.0491127967834473, 1.5726701021194458, 0.007268615998327732, -0.6977748870849609, 0.1449267417192459, 0.41879990696907043, -0.05005645379424095, -0.07730825990438461, 0.5061310529708862, 1.4381884336471558, 1.7035276889801025, 1.707472562789917, 1.9734963178634644, 0.2642926573753357, -1.0013560056686401, 0.3647702932357788, -0.27899065613746643, -0.8700121641159058, -0.8818623423576355, -0.062142521142959595, 0.4097520411014557, -0.34951168298721313, -0.3423663079738617, -0.1564975380897522, 0.4883776009082794, -0.35016441345214844, -2.2098212242126465, 1.7042791843414307, 0.3836303949356079, 0.6911745071411133, 0.7943276166915894, 0.015778955072164536, 0.5861324667930603, -0.36225658655166626, 2.3749327659606934, -0.8035104870796204, 1.734454870223999, -0.1374453753232956, -1.3536862134933472, -1.9125417470932007, 0.06854409724473953, -2.071833848953247, 0.25186869502067566, 1.51982843875885, 1.3510072231292725, 0.5429275631904602, 3.2342395782470703, 0.7839298248291016, -0.17015719413757324, 0.5731907486915588, -0.4185318052768707, 0.2406935840845108, -0.5762285590171814, -0.78608638048172, 0.051906850188970566, -1.2581368684768677, -1.4168070554733276, -1.0982003211975098, 0.34494534134864807, -1.1745737791061401, -0.8407520055770874, 0.6580320000648499, -1.457546353340149, 0.8845716118812561, -0.8369802832603455, -0.8013685345649719, -0.0024289304856210947, 0.10661010444164276, 0.7509618401527405, 0.6132997274398804, 1.1314527988433838, 3.1594486236572266, -0.8475711345672607, 0.15657897293567657, 0.006533011794090271, 0.5023981928825378, -1.4584156274795532, -0.5861316323280334, 1.563982605934143, -0.5970779657363892, -0.3221726417541504, -1.4166699647903442, -0.15515488386154175, -0.35658732056617737, -0.39959946274757385, 1.710015892982483, 0.9837565422058105, -0.9022293090820312, -1.0723997354507446, 1.420004963874817, 0.4694691002368927, -1.8641396760940552, 0.8212056159973145, 2.201352119445801, -0.7393127083778381, -1.5246362686157227, -1.9605624675750732, -0.4119507968425751, -0.06326530873775482, 0.007366941310465336, -0.5217618942260742, 0.6024919152259827, 0.3506562113761902, 0.8054545521736145, -0.07937243580818176, -0.13712048530578613, -0.3888816237449646, -0.3143916130065918, -0.46081987023353577, 0.3032538890838623, -2.2426419258117676, -0.005326972808688879, -0.05277285352349281, 0.17818424105644226, 0.7821739912033081, 0.9502089619636536, -2.221076250076294, -0.7651772499084473, -1.5838836431503296], [1.5744551420211792, 0.326322078704834, -0.2955315113067627, 1.0506641864776611, -0.777914822101593, 0.12964555621147156, 0.8184725642204285, 0.8449187278747559, 0.6136152148246765, 1.0202723741531372, -0.016348714008927345, 0.2994694709777832, 0.35547879338264465, 0.2882065176963806, 1.2929667234420776, -0.1131812185049057, -1.0843278169631958, 0.8008168935775757, 0.6964081525802612, 0.49252834916114807, -0.14291511476039886, -0.24914225935935974, 0.5907509922981262, -0.5734383463859558, -0.017839858308434486, 0.9078859090805054, 0.4768458902835846, 0.8488999605178833, -1.1909645795822144, -0.9295316338539124, 1.024031162261963, 1.222578525543213, 0.04963766410946846, 0.855888307094574, 0.17614129185676575, 1.8867439031600952, -1.5129255056381226, 2.2102649211883545, -0.5030085444450378, -0.4159446060657501, 0.42286738753318787, 1.0079007148742676, 0.792568564414978, -0.2735908329486847, -1.4145864248275757, 0.8955830931663513, -0.03247992321848869, -0.02649000473320484, 3.0754220485687256, -0.8655933737754822, 1.0836329460144043, 0.1605217158794403, -1.265743374824524, 0.4700445532798767, -0.5371273756027222, 0.31239771842956543, -0.2833155691623688, -0.892737627029419, 0.15887373685836792, -0.6098835468292236, -1.5016121864318848, -0.2650240957736969, -0.8309324979782104, 0.2510136365890503, -0.06244318187236786, 0.09459275007247925, 0.5992065668106079, 0.31877362728118896, 1.8251426219940186, -0.8604645729064941, -0.09755405783653259, -1.1990928649902344, -1.188550591468811, -0.1071336418390274, 2.2109100818634033, 0.5822893381118774, 0.505736768245697, 0.4528197646141052, -0.769309401512146, -0.1027049571275711, 1.3683375120162964, -0.28034859895706177, 1.0543813705444336, -0.6420798897743225, -1.6398367881774902, -1.103550910949707, 1.3768750429153442, -0.8810907602310181, -0.2859458923339844, -0.16046380996704102, 2.121628761291504, 1.8258854150772095, -1.2579283714294434, -1.0631098747253418, 0.5377939939498901, 1.1299140453338623, -1.385853886604309, -0.10115911066532135, -0.636022686958313, -0.3038993775844574, -1.3902441263198853, 0.36162567138671875, -1.5280487537384033, 0.29571807384490967, 0.3578559458255768, 0.04101342707872391, -0.7404640316963196, 1.8155242204666138, 1.2528038024902344, 0.34709206223487854, -0.900616466999054, -1.962794542312622, -0.18948014080524445, -1.2797596454620361, -1.1432874202728271, 0.7578368186950684, 1.1210583448410034, 0.5735321640968323, 0.07320179790258408, 1.5728758573532104, -0.6512284874916077, 0.6538901329040527, -0.09573721885681152, 1.4410821199417114, -0.9658191204071045, 1.9099727869033813, 0.0417441762983799, 0.3705310821533203], [-0.30450573563575745, 0.29794418811798096, 0.14390112459659576, -1.8932807445526123, -0.4430210292339325, 1.1893962621688843, -0.3486909866333008, -0.14356543123722076, 0.38229623436927795, -0.9587192535400391, -1.2266643047332764, -0.5139713883399963, -0.4762570261955261, 2.061560869216919, 2.5992279052734375, 0.7560071349143982, -0.3706015348434448, -0.7544931173324585, -0.9699856638908386, -0.73212069272995, 0.5577508211135864, 0.16669277846813202, -0.5951476097106934, -0.25519782304763794, 0.9512012600898743, 1.1944093704223633, 1.4406991004943848, 2.167651891708374, -0.8763169050216675, 0.5990950465202332, -0.9859153032302856, -0.7693026065826416, -1.3141324520111084, 0.535720705986023, 0.08479390293359756, 0.6378691792488098, 1.3362524509429932, 1.4927997589111328, -0.47234195470809937, -0.23518656194210052, -1.2724472284317017, 1.10409677028656, 0.4705972671508789, -0.044582549482584, 0.43300291895866394, 1.6467424631118774, 0.669903039932251, -0.734072744846344, 1.8309054374694824, 0.04223038628697395, -0.2752455472946167, 0.12231907993555069, 1.674232840538025, 0.3497849106788635, -0.421130895614624, -0.8777462840080261, -0.1557343602180481, 1.0408704280853271, -1.3301321268081665, -1.6228222846984863, 0.15477916598320007, -2.0072431564331055, -0.540059506893158, 1.1323559284210205, 0.09076395630836487, -0.5479495525360107, -0.4876367449760437, -1.0936918258666992, -1.2493314743041992, 0.3804599642753601, 0.43485385179519653, 0.4517175257205963, -0.550821840763092, -0.49837470054626465, 0.3851347267627716, 0.36246457695961, 0.5010454058647156, 1.831575632095337, -0.9826905727386475, 2.384458065032959, -0.1614135354757309, 0.7848504185676575, 2.150204658508301, -1.7666947841644287, -0.40431728959083557, 1.2862476110458374, -0.8431171774864197, -0.6608656048774719, -0.08144226670265198, 1.1673699617385864, -0.33645087480545044, -0.32280951738357544, -1.1819608211517334, 1.9838005304336548, -0.9008405804634094, -1.1345638036727905, -0.1163523942232132, 0.25148653984069824, 0.11739913374185562, -0.128273606300354, -1.9200161695480347, 1.2695722579956055, 1.5546644926071167, 1.1556179523468018, 0.8975890874862671, -0.09622253477573395, -0.35817036032676697, 0.6882736086845398, 0.976237416267395, 0.8543545603752136, -1.28016996383667, 1.0238933563232422, -1.4011189937591553, 1.5309138298034668, 1.7506113052368164, -1.8233013153076172, 2.222055196762085, -1.252905249595642, -0.0044790152460336685, 0.6845923066139221, -0.6253058910369873, -0.3300445079803467, -1.715815782546997, -0.007593974936753511, -0.13624213635921478, -0.09226715564727783, 0.7551908493041992, 1.0171207189559937], [0.23010191321372986, -1.6687257289886475, 0.997017502784729, 1.077807903289795, 0.8351292610168457, -0.6191691160202026, -1.274810552597046, -0.8480709791183472, 0.9609830379486084, 0.6937135457992554, -0.9399760365486145, 0.9691116809844971, 1.4772087335586548, 0.5209885239601135, 0.5615431666374207, -0.17573830485343933, -1.0684418678283691, -0.5644872784614563, 1.1583771705627441, 1.186293601989746, 1.2434697151184082, 0.7102512121200562, -1.2808741331100464, -0.7852211594581604, -1.4392637014389038, -0.35205402970314026, -1.649383544921875, -0.7347331643104553, 1.046146035194397, -0.6777902245521545, -0.05733728036284447, -0.06339731067419052, 0.27182725071907043, 0.4881806969642639, 0.7609742283821106, -0.8568674325942993, 0.9225348234176636, 1.555106520652771, -1.016638159751892, -1.058915138244629, -0.9255058765411377, 1.3136688470840454, -0.43730229139328003, 0.17679093778133392, -0.18856358528137207, -0.8018760085105896, 0.2506301701068878, -1.475175380706787, 0.8105257153511047, -0.9730983376502991, 0.5759010910987854, 0.9200031161308289, -0.803188145160675, -1.683448076248169, -0.49234631657600403, 1.4629642963409424, 0.6908409595489502, -0.8185417056083679, 0.9340969324111938, 1.0817514657974243, -0.5432102084159851, -1.0280702114105225, 0.8171625137329102, 0.6526238918304443, 1.1232693195343018, -0.4973348379135132, 0.04848852381110191, -1.6813853979110718, 1.1430577039718628, -1.3718085289001465, 1.4246201515197754, -0.14741453528404236, 2.266279458999634, 1.7056386470794678, 0.951178789138794, -1.337372899055481, 0.26863405108451843, 1.1481027603149414, 3.4769701957702637, -0.1454010307788849, -0.6293060183525085, -2.7470576763153076, -2.5228617191314697, 0.46851587295532227, -0.6699427962303162, 0.9509104490280151, -0.8195191621780396, -0.3034372925758362, 0.25456592440605164, 0.3666161894798279, 0.20281916856765747, 0.7550346255302429, 0.254512757062912, 0.787360429763794, 0.7196658253669739, -0.8521923422813416, 0.05234192684292793, -0.7696577906608582, 0.6773847937583923, -0.6936976313591003, -0.3823626637458801, 0.4868534505367279, 0.6902714967727661, 0.33391132950782776, -2.085865020751953, -1.4746569395065308, 0.7380182147026062, -0.1438959240913391, 1.4086170196533203, 0.37287068367004395, -2.0489399433135986, -1.193385362625122, 1.3131775856018066, 1.0243245363235474, -0.30561721324920654, -1.4451385736465454, -0.13455522060394287, -0.8467423319816589, 1.6773970127105713, 0.6883530616760254, 1.2096308469772339, 0.7750425934791565, -0.27620548009872437, 0.8531391620635986, 0.12111376225948334, -0.8782044053077698, 0.4965544641017914, 0.07636857777833939], [-1.5746400356292725, 0.11571560055017471, 0.9139714241027832, -0.07835329324007034, -0.41700008511543274, -1.00569748878479, -0.6253987550735474, 2.4224932193756104, 0.3916538953781128, 0.4322451949119568, -1.8049670457839966, 0.3472154438495636, -0.8621456623077393, -0.310051292181015, -0.8443114161491394, -0.7400732040405273, -1.0683550834655762, -1.074454665184021, 0.739172637462616, 0.0014518112875521183, 1.4725528955459595, 0.21599049866199493, 0.3555358648300171, 1.2746026515960693, -1.0296822786331177, -0.20269571244716644, 0.7140375375747681, -1.751621127128601, -1.1674888134002686, -0.06407657265663147, 0.4253961145877838, -0.67216557264328, -1.2305233478546143, -0.04808228835463524, 1.5934488773345947, 0.9993169903755188, 0.1216580718755722, -0.3286643922328949, -0.8516234159469604, 0.804190456867218, -0.11200138926506042, -1.7202903032302856, 0.5833657383918762, 0.8521655201911926, 1.453467845916748, 1.37328040599823, -0.2362101972103119, -0.48839274048805237, 0.6419246196746826, -1.9874396324157715, -0.2758593261241913, -0.8971683979034424, 0.43507304787635803, 1.7962586879730225, -0.4179791212081909, -0.5748420357704163, -0.4063885807991028, -0.5225121974945068, 0.058638427406549454, 2.231076955795288, 0.9445862174034119, 0.26335132122039795, 0.19592680037021637, -0.35226643085479736, 0.6643351912498474, -1.1845391988754272, 1.0824376344680786, 0.34013858437538147, 0.1946234107017517, -1.3529980182647705, 1.0381981134414673, 0.4695356786251068, 0.08584363758563995, -0.32253846526145935, 0.49306654930114746, 1.9650532007217407, 0.26732078194618225, 0.47964242100715637, -2.023160696029663, -0.018241411074995995, 1.4380700588226318, -1.2479047775268555, 0.24226883053779602, -1.9418718814849854, 1.7698100805282593, 0.8298547863960266, 0.20263977348804474, -1.6666678190231323, 0.9524551630020142, 1.641109585762024, -0.6926088333129883, 0.782387912273407, 0.29500314593315125, -0.3489394783973694, -1.4731197357177734, 0.3644426763057709, -0.17681224644184113, -0.5854448676109314, -0.6380172967910767, -0.881285548210144, -1.750284194946289, -0.9019474387168884, 1.3525357246398926, -1.9151262044906616, 1.2048100233078003, 0.8506922721862793, 1.1609433889389038, 0.6255161762237549, 1.1847500801086426, -0.8184753060340881, 1.385986328125, 0.21746350824832916, -1.2798064947128296, 0.8128321766853333, -0.9022789001464844, 2.3148505687713623, 1.873016357421875, 0.8119331002235413, 0.9616644382476807, 0.7447817921638489, -0.23364204168319702, -0.5087809562683105, -0.32728034257888794, -1.3097929954528809, 1.4410400390625, 0.45840924978256226, -0.7636802196502686, 1.7062283754348755], [-1.5217863321304321, 0.9737792015075684, -0.8636108040809631, 2.1973776817321777, -1.2882654666900635, -1.3497025966644287, 0.34795862436294556, 0.2717815041542053, 0.4177095890045166, 0.6118646860122681, -0.47779637575149536, -1.1351944208145142, -0.015610642731189728, 1.6331958770751953, -0.5392769575119019, -0.6756677031517029, 0.8088034987449646, -0.3935442566871643, -1.6172406673431396, -1.933736801147461, -0.9472312927246094, 0.8348308205604553, 1.016527771949768, 0.24092429876327515, -1.1586239337921143, -0.6766147017478943, 1.4590259790420532, 1.6829911470413208, 0.6545968651771545, 0.0025502718053758144, -0.16030475497245789, -0.6162059307098389, 0.0347440131008625, -1.7591947317123413, -0.80122971534729, -0.2885020077228546, 1.1435651779174805, -1.209040880203247, 0.468587726354599, -1.2361916303634644, 0.6266922950744629, -1.8298872709274292, 0.47426068782806396, 0.2729892432689667, 1.153691053390503, -1.224288821220398, -0.8136333227157593, 0.338199645280838, -0.16753292083740234, -0.8433467745780945, -0.01754923164844513, 0.13847312331199646, -1.5626500844955444, -0.1655423790216446, 1.837568759918213, -2.229172468185425, -0.560771644115448, -0.2410982996225357, -0.09828058630228043, 0.139382004737854, 0.6683362722396851, -0.3829216957092285, -0.40779319405555725, 1.3090397119522095, 0.374514102935791, -0.7698658108711243, -0.3613017499446869, -0.27665823698043823, 0.7054963111877441, -1.065793752670288, -2.0393199920654297, 0.33928608894348145, -0.9142195582389832, -1.5752863883972168, 1.4975472688674927, 0.09771372377872467, -1.7541617155075073, 0.7372218370437622, 0.18621183931827545, -0.3980032205581665, 0.13207575678825378, 0.23878948390483856, 1.5940889120101929, 0.30975541472435, 0.7122834920883179, 0.4339836537837982, 0.239213228225708, 1.7842389345169067, 0.5889021158218384, -1.1250264644622803, 0.07645749300718307, 1.2240626811981201, 1.7685168981552124, -1.2324042320251465, 1.2640067338943481, 1.4036649465560913, 0.30639591813087463, 0.6672641038894653, -0.8303453326225281, 1.6579290628433228, -0.439087450504303, -1.0653235912322998, -0.07949218153953552, -0.6083402633666992, -0.37336820363998413, -0.4598625600337982, -0.5733242630958557, 0.26304903626441956, -0.2922143340110779, -0.380429744720459, -0.23923546075820923, -0.4775873124599457, 1.0476934909820557, 0.28099584579467773, 0.36327674984931946, -0.11170625686645508, 0.574432373046875, -2.18257999420166, 0.04248204454779625, 1.4522161483764648, -0.28956255316734314, -0.1850472390651703, 0.793393075466156, -0.027049293741583824, -2.1848838329315186, 0.39947396516799927, -1.7509125471115112, 1.2409669160842896], [1.000726342201233, -0.24005872011184692, 0.13845527172088623, -0.3621155023574829, -0.9228251576423645, -0.24074995517730713, 0.9910323023796082, -0.2795618176460266, 0.40985068678855896, -0.976173460483551, 1.4255450963974, 0.6365911960601807, 1.2398643493652344, -1.0057636499404907, 0.39931201934814453, -0.06450306624174118, 0.04086737707257271, -1.0394402742385864, 0.33998236060142517, 0.32542189955711365, 0.36772555112838745, 0.6115315556526184, 1.9916586875915527, -0.9209679365158081, 0.43579623103141785, 0.2211899757385254, -0.2959357500076294, -0.5882900953292847, -1.0199705362319946, 1.128003478050232, -0.5774866938591003, 0.12038545310497284, -0.1452924609184265, -0.3450075089931488, -0.12329386174678802, 0.3284424841403961, 1.1542742252349854, -0.22431224584579468, 0.44906172156333923, 0.6348806023597717, 0.592236340045929, -0.11068940162658691, -0.9766237139701843, -0.6322082877159119, -0.3158092200756073, -0.8920300006866455, 0.5976595282554626, 0.764268159866333, -0.7972515821456909, -2.0737340450286865, -0.03179575875401497, 0.7519746422767639, -0.22609122097492218, -0.29576584696769714, -0.9534205198287964, 1.1866841316223145, -0.11100085079669952, -1.2990490198135376, 0.4587451219558716, 0.6714625358581543, 1.6710519790649414, -0.40978264808654785, 1.3768469095230103, 0.4154681861400604, -0.21451832354068756, -1.3141998052597046, -1.8192065954208374, -1.2331262826919556, -0.9029651284217834, -0.02246328629553318, 0.7673593759536743, 2.0685582160949707, -0.7229813933372498, 0.5204745531082153, 0.14627064764499664, 0.7359887957572937, -1.8282414674758911, 0.9559792876243591, -0.7711055874824524, -2.1878366470336914, -2.729656219482422, -0.4432661533355713, 1.2956963777542114, 1.1582237482070923, 1.1591695547103882, -0.5213496685028076, -0.07730372250080109, -1.4558993577957153, -0.8476482629776001, 0.07532288134098053, -1.1637402772903442, -0.5830276608467102, 1.0031802654266357, 0.3656845986843109, -2.2260613441467285, 2.733058214187622, -0.12468922883272171, -0.22637920081615448, 0.034620098769664764, 0.24612492322921753, -0.5724431872367859, 1.1966837644577026, -1.2087860107421875, 1.7260456085205078, 0.015324492007493973, -1.0359855890274048, 0.4978273808956146, -0.45495545864105225, -0.3251184821128845, 0.20422637462615967, -0.9605815410614014, -0.23801495134830475, 0.7361930012702942, 0.9235918521881104, -0.08106134831905365, 0.9635725617408752, -0.08858629316091537, 0.3364247679710388, 0.6260611414909363, -0.6515377163887024, -0.3068310022354126, 1.5049957036972046, 0.07203736156225204, 1.1703696250915527, -2.4374687671661377, 0.15115313231945038, -3.064161539077759, 0.735441267490387], [-1.2772542238235474, -0.6004554629325867, -0.36695441603660583, 0.5984697341918945, 0.05582484230399132, -0.28223446011543274, -0.3677375614643097, -0.9739097952842712, 1.0614680051803589, 0.12468903511762619, -1.6872345209121704, 0.5350081324577332, 0.30732694268226624, -1.3624498844146729, -0.8578709363937378, -0.0874275341629982, 0.23754385113716125, 0.6307183504104614, 1.9347710609436035, 1.201821208000183, 0.4630102813243866, 0.3157840073108673, -0.53549724817276, 0.21306362748146057, 0.8645868301391602, -0.48862653970718384, 0.22056233882904053, -1.4193756580352783, 0.26673126220703125, -0.4138535261154175, -1.9607824087142944, 0.5503795742988586, -0.5038363337516785, -0.15927591919898987, 1.3564485311508179, 0.3154430687427521, 0.8235997557640076, -0.026910122483968735, 1.0497040748596191, 0.02828400395810604, 0.3990265130996704, -1.5616871118545532, -0.559369683265686, -1.0516090393066406, 1.2819024324417114, -1.4305741786956787, 1.0584564208984375, -0.9128793478012085, -0.031055418774485588, -0.533944308757782, 0.2726038694381714, -0.012202592566609383, 1.9846957921981812, -0.6497330665588379, 1.652453899383545, -0.9756048917770386, -2.0465340614318848, 0.4115375876426697, -0.5543524622917175, 0.3551565408706665, -0.17701393365859985, 0.47590371966362, 2.440863609313965, -0.09596127271652222, 0.8180685639381409, 1.2623854875564575, -0.44468772411346436, -0.736576497554779, -0.9146571159362793, 1.274010181427002, -1.6902247667312622, -0.45080411434173584, 0.31931912899017334, 0.5179874897003174, 1.9282052516937256, 0.49476414918899536, 1.4451302289962769, 1.101428508758545, 0.1300099641084671, -1.8032996654510498, -1.3789747953414917, -0.4628024995326996, 0.26384928822517395, -1.1588383913040161, -0.4829900860786438, -0.6895598769187927, -0.3603284955024719, 0.6606153845787048, 1.5568751096725464, 0.2669486701488495, -0.6412111520767212, 1.3484513759613037, -1.166346549987793, -1.927065372467041, 0.10052265971899033, -1.0953705310821533, -0.025919470936059952, -1.0380626916885376, 1.6299285888671875, -0.7991737127304077, -0.16735993325710297, -0.289323091506958, 0.788162112236023, 0.5339588522911072, 0.7275087833404541, -0.5501753091812134, -0.0354209765791893, -0.5937364101409912, -0.7679827213287354, -0.05184876173734665, 0.8167505264282227, 1.3100008964538574, -0.7350267171859741, 0.4548266530036926, -1.0768120288848877, 0.39479678869247437, 3.177016019821167, 0.3565244674682617, -1.0154180526733398, -0.11328206956386566, 1.6101481914520264, 0.6960600018501282, -0.17418842017650604, -0.3323419392108917, -0.6248112916946411, -1.0966591835021973, -0.7596156597137451, 0.6788817048072815], [0.22069084644317627, 1.1227779388427734, -1.6121655702590942, -0.3935360312461853, -0.3192864954471588, -1.0435941219329834, 0.3536035418510437, -1.885079264640808, 0.23135590553283691, -0.7574580907821655, -1.2860864400863647, -1.4832215309143066, -0.425703763961792, 0.10481172800064087, -1.5425180196762085, 0.40630048513412476, 0.8484519720077515, -2.3221776485443115, 0.7891584634780884, 0.8618212342262268, 0.016684554517269135, -1.6716628074645996, -0.8995975255966187, -0.8703266382217407, 0.7448345422744751, 1.0797942876815796, 0.970641016960144, -0.019002137705683708, 0.9459446668624878, -0.7445005774497986, -0.48893043398857117, -1.9496561288833618, 0.5969160795211792, 0.24449950456619263, -0.13822691142559052, -1.508657693862915, 0.15063747763633728, -0.017298787832260132, -1.4906507730484009, 0.7018209099769592, -0.5477002263069153, 1.4481003284454346, 0.5733679533004761, -0.2931828796863556, -1.530636191368103, -0.03464854136109352, 0.12895749509334564, -0.4421267807483673, 0.47805166244506836, 0.29481396079063416, 1.2121667861938477, 1.7255016565322876, 0.42349979281425476, -0.08799952268600464, 1.9912476539611816, 1.5016429424285889, 0.2809780240058899, 1.7806520462036133, 0.28055840730667114, 0.8195304274559021, -0.5251573920249939, -0.7477943301200867, -1.0920790433883667, 0.5204623341560364, -0.7667753100395203, 1.11873459815979, 0.39170724153518677, 0.3617919683456421, -0.7464669346809387, -0.37995970249176025, -0.7460061311721802, 0.9257418513298035, 2.572763442993164, 1.3212945461273193, -1.2697083950042725, -0.3247450292110443, 0.6987540125846863, -2.156569480895996, -0.246599942445755, 0.9432148933410645, 0.17879116535186768, 0.1698647439479828, -0.18839110434055328, 1.318956971168518, 0.12768760323524475, 1.936170220375061, -0.9479113817214966, -0.08236677199602127, -0.7612722516059875, -1.3602821826934814, -0.22563916444778442, 0.837013840675354, -0.30898818373680115, 1.0516691207885742, 0.7305186986923218, -1.4849711656570435, 0.19273068010807037, -0.21516892313957214, 0.6502334475517273, 0.31733888387680054, 0.26853513717651367, -0.46700212359428406, -0.9167325496673584, 0.514087975025177, -0.5263895392417908, 1.2442289590835571, -1.4310753345489502, -0.6385411620140076, -1.316627025604248, 1.3307716846466064, -0.6570228338241577, 0.35307732224464417, -2.0653011798858643, 0.005085202399641275, -1.8570085763931274, 0.11327669769525528, 1.3836029767990112, -0.17217093706130981, -0.19602876901626587, -0.627919614315033, -0.23772013187408447, 1.391737937927246, -0.7689473032951355, -0.1851576864719391, 0.9268079400062561, 0.5712351202964783, -0.11004313826560974, -1.1243399381637573], [-1.0364384651184082, -0.4104169011116028, -0.2502953112125397, -1.2525019645690918, 0.7868131995201111, -0.9106033444404602, -1.6627798080444336, -0.9408959746360779, -0.41025620698928833, 1.5564699172973633, -0.14354869723320007, 1.7344354391098022, 0.22044281661510468, 0.07702808827161789, 0.4173220098018646, 1.243456244468689, 0.018552420660853386, 1.0940463542938232, 0.6110369563102722, -0.4590211808681488, -0.4806195795536041, 0.05722663924098015, -0.2003050148487091, 0.1362120807170868, 0.5338728427886963, 0.16714558005332947, -0.8258984088897705, 0.7084224820137024, -0.9077085256576538, 0.27506816387176514, -0.7828643321990967, 0.21011944115161896, 0.41686999797821045, 1.6841894388198853, 0.8031035661697388, -0.8584267497062683, 0.6205337643623352, 1.5250334739685059, 0.08142544329166412, -0.28708595037460327, 0.7503785490989685, -0.34183529019355774, -0.4023939073085785, 1.1549451351165771, -0.3468695878982544, -0.6274203658103943, 0.637144148349762, 0.33902305364608765, 0.3217643201351166, -2.0099313259124756, -0.6182420253753662, -0.7036944031715393, 2.1617939472198486, -0.744566798210144, -0.3905928134918213, 0.6532055139541626, -0.21870429813861847, -0.275589257478714, -0.5884777903556824, 1.4467129707336426, -0.6090418100357056, 0.26408979296684265, -0.47606393694877625, 0.9820674061775208, -2.323016405105591, 0.5469964146614075, -1.6595079898834229, -0.9213458299636841, 0.5379348993301392, -1.163611888885498, -0.9491575956344604, 1.070426106452942, -0.23007293045520782, -0.8908267021179199, -1.3441739082336426, -1.0845530033111572, 1.0677452087402344, -1.2194501161575317, -0.5874239802360535, -1.4473402500152588, 0.03279675915837288, 0.9949619174003601, -1.2992346286773682, 0.9153295755386353, 0.5782473087310791, -1.6649329662322998, -0.32523587346076965, -0.3052000105381012, 0.5633582472801208, -0.18011963367462158, -1.6164263486862183, 1.152004599571228, -0.8832975625991821, -0.14298103749752045, -0.04052416980266571, -1.6181846857070923, -0.39527592062950134, 1.8857859373092651, 0.36948034167289734, -0.3311631381511688, 0.7418729662895203, -1.1825870275497437, 0.01494542695581913, 1.5393414497375488, -1.9070345163345337, -2.0934951305389404, 0.6043210029602051, 2.7148211002349854, 0.18886195123195648, 0.501469612121582, -0.8042840361595154, -1.101264238357544, 1.4975895881652832, 2.9025862216949463, -1.0437554121017456, -0.009246909990906715, 0.8554442524909973, 1.2057844400405884, -1.3850017786026, -0.7516359090805054, -0.8996748924255371, 0.8700974583625793, 0.7331832647323608, -0.3053258955478668, 1.5738461017608643, -1.0540634393692017, -0.36914294958114624, -0.1977596879005432], [1.7093896865844727, 0.17762641608715057, -1.6638892889022827, -0.8283787369728088, -0.47937142848968506, -0.21089650690555573, 1.3852745294570923, -1.265601634979248, 0.1664281040430069, -0.8337776064872742, -0.2512521743774414, -0.5761017799377441, 0.4531947374343872, -1.0234578847885132, -0.48684003949165344, -1.1745151281356812, -0.33418744802474976, -1.516372561454773, 0.37195026874542236, 0.1303664743900299, 0.2614078223705292, 1.113566279411316, -0.7285796999931335, 1.3026307821273804, -0.8155444264411926, -2.155064344406128, -1.0418412685394287, 0.2614802420139313, 1.123289942741394, -1.4560520648956299, -0.304948627948761, -0.2413598895072937, 0.215408056974411, -1.4735925197601318, -0.5530213713645935, 0.19002172350883484, 0.8820708990097046, -2.4265451431274414, 1.1244481801986694, 0.013409068807959557, -0.5529003739356995, -1.2714673280715942, -0.08137746900320053, 0.4349002242088318, 0.4390483498573303, -0.9488611221313477, 0.40390753746032715, -0.00458852294832468, -0.15829181671142578, 0.9980976581573486, 2.3562066555023193, 1.1083202362060547, -0.9678314924240112, 0.5365726351737976, -0.7410812973976135, -2.758152484893799, -0.16956007480621338, -0.6030994653701782, 0.8332604169845581, -0.8749139904975891, 0.8898388147354126, -1.2574504613876343, -1.5992035865783691, 1.4489554166793823, -0.05080693960189819, 0.12555699050426483, -0.2563224732875824, 0.3294675052165985, -0.8223864436149597, 0.9377129673957825, 1.1973650455474854, 0.40041646361351013, -0.5468509793281555, -0.17072191834449768, 0.7518814206123352, 1.1911605596542358, -0.5659798979759216, 0.989707887172699, 0.08098214864730835, -1.044023871421814, 0.7071458697319031, 1.543013095855713, 0.07775463908910751, 0.6181771755218506, -0.06738745421171188, -1.1722378730773926, 1.0637195110321045, 0.72988361120224, -0.7988496422767639, 1.8538583517074585, 1.2973154783248901, -1.192906141281128, -1.0519237518310547, -0.38691672682762146, 1.5810388326644897, -0.46422386169433594, -0.920612633228302, -1.8047319650650024, 0.21278025209903717, -0.9305791854858398, 0.7512266635894775, 1.8207048177719116, -0.17740489542484283, -1.4606484174728394, 0.09372828155755997, 1.4233882427215576, -0.32528167963027954, 0.29362815618515015, -0.6366463899612427, 0.6849784255027771, -0.09752984344959259, 0.12275783717632294, -0.6267498135566711, -0.42484188079833984, -0.5707153081893921, -0.2911953032016754, -0.40922072529792786, 0.1006070151925087, 1.4517048597335815, 1.0799505710601807, 0.31824901700019836, -0.8808591961860657, -0.19514669477939606, -0.09308173507452011, 2.277710437774658, -2.0722949504852295, 0.16462166607379913, -1.3922888040542603], [-0.8403884768486023, 0.11809840053319931, -0.04731703922152519, 2.348539113998413, 0.872705340385437, -0.7111290693283081, -0.012487152591347694, -0.7673301696777344, 1.362040400505066, -0.25180143117904663, 1.4333418607711792, 1.4669129848480225, -3.6658859252929688, -0.2035481333732605, 0.6380649209022522, -1.2287863492965698, 0.4370046555995941, -0.24010345339775085, 0.20064912736415863, -0.5471188426017761, 0.25642406940460205, 1.4672666788101196, -0.1871812492609024, 1.3387645483016968, -0.10183914750814438, -1.7676236629486084, -0.7190889716148376, -0.8408563137054443, -1.3077040910720825, -0.7071720957756042, 2.131293296813965, -0.3644091486930847, 1.9077085256576538, -0.37044090032577515, -0.48224350810050964, -2.138446569442749, 0.3833088278770447, 0.43024569749832153, 0.3018610179424286, 0.14251647889614105, -1.3877071142196655, 1.2413405179977417, 1.4947649240493774, 0.015230047516524792, -0.5466383695602417, -0.5231532454490662, 0.9959595203399658, -0.4877004325389862, -1.1042022705078125, -0.6920278072357178, -0.49214890599250793, 0.689348578453064, 1.6296308040618896, -0.18596772849559784, 0.8996768593788147, 0.128932923078537, -1.0370848178863525, 0.7675752639770508, 0.22859302163124084, -1.0723199844360352, -0.47302308678627014, -0.45838242769241333, 0.1871953159570694, -0.1424056738615036, -0.4285300374031067, -1.4432579278945923, -0.17855045199394226, 0.3431466817855835, -1.3141376972198486, 0.4477895498275757, 1.1186096668243408, -0.2796369194984436, -0.5524203181266785, 0.15789368748664856, 0.9040833115577698, -0.22767993807792664, 0.6905876994132996, 1.7594534158706665, 1.024214267730713, 0.8220387101173401, 0.004419934935867786, -0.9582284688949585, -0.4096059203147888, 0.1517903059720993, -0.2736327052116394, -0.1008412167429924, -0.6374163031578064, 0.6336201429367065, 0.12824469804763794, -0.44684094190597534, 0.1656630039215088, 1.1246346235275269, 2.0578043460845947, 1.1539604663848877, 1.2887537479400635, 0.9436708688735962, 1.6802520751953125, -0.7303187847137451, 1.6889415979385376, 0.9114269018173218, 1.0173879861831665, 0.536426305770874, -2.073625087738037, -0.029545791447162628, -0.2903314530849457, -0.36696290969848633, -1.2015377283096313, -0.7192136645317078, 1.3313148021697998, -1.0829013586044312, 0.5756344795227051, -0.7792432904243469, -2.4212207794189453, 0.8495041728019714, 1.3408437967300415, -1.6059037446975708, 1.188924789428711, -1.5669975280761719, -1.046701431274414, -0.7396310567855835, 0.8482450842857361, -0.018274866044521332, 0.05683941766619682, 0.673904538154602, -0.30628958344459534, 0.44765785336494446, -0.847792387008667, 1.3397740125656128]]]], "dtype": "float32"}, {"__nd__": [[[[0.8736039996147156, 0.15731936693191528, 0.6875300407409668, -0.12005700170993805, -0.7395879626274109, -0.2597513198852539, -0.7716795206069946, -1.2318142652511597, 0.2216801792383194, 0.013803155161440372, -1.9095252752304077, -0.7770386338233948, -0.685000479221344, 0.006419667508453131, 1.675560474395752, -0.5384746193885803, -0.6377695202827454, -0.6754165887832642, 0.5895214676856995, 1.046329140663147, -0.31823837757110596, -0.6339491009712219, 0.4911782145500183, -0.37347137928009033, 0.8282777070999146, -1.703218698501587, -0.31221815943717957, -0.0243344996124506, 0.07835352420806885, -0.6220927238464355, 0.31472596526145935, 0.9638266563415527, -0.27920153737068176, -0.7138554453849792, 0.34674766659736633, 0.7846325039863586, -0.0991014614701271, -1.8570845127105713, 1.2253729104995728, -0.8912966251373291, -0.271480530500412, -2.438021659851074, 0.6858837604522705, 0.8134121894836426, -0.20889516174793243, -0.7200455665588379, 1.2029260396957397, -0.25584572553634644, -0.9550201892852783, -0.21053285896778107, 0.5546116232872009, -1.0595520734786987, 0.6627993583679199, -1.1942813396453857, -1.488598108291626, 0.5653240084648132, 1.1102819442749023, -1.3289159536361694, -1.9398627281188965, 1.8079006671905518, -0.5966861844062805, -1.1470540761947632, 0.7123070359230042, 0.025793297216296196, 0.16077691316604614, -0.3601173460483551, -1.2457104921340942, -1.1827094554901123, 0.9613455533981323, 0.7189039587974548, 0.9330087304115295, 0.05873674526810646, 0.6094839572906494, 0.24679623544216156, -0.3582583963871002, 0.7225037217140198, -0.47776153683662415, 0.027546389028429985, -2.1546576023101807, -0.045462850481271744, 0.07580691576004028, 0.061164166778326035, -1.4012675285339355, -0.43210652470588684, -1.495564341545105, -0.5231676697731018, -0.44172191619873047, -0.5283340811729431, -0.9000887870788574, -0.20933528244495392, 1.6438946723937988, 0.18106037378311157, -1.0310934782028198, 1.669778823852539, 0.5008178949356079, 0.028434157371520996, -2.245677947998047, 1.3082104921340942, -1.3940831422805786, 0.5710281133651733, -1.5829732418060303, 0.9836093187332153, -0.1659993976354599, -0.23785588145256042, 1.2592928409576416, -1.0423541069030762, -1.0450700521469116, -0.7531119585037231, 0.197585791349411, -0.8500763177871704, 0.369705468416214, 1.1607835292816162, -2.2201802730560303, 0.8130172491073608, -0.2151542752981186, -0.5665802359580994, -0.19740906357765198, 0.5671038627624512, -0.3805658519268036, -0.5157858729362488, -0.592960000038147, -0.39040493965148926, 0.2767347991466522, 0.9752312302589417, -1.9994666576385498, -0.6698939800262451, 1.1935036182403564, -0.4925214350223541], [0.6824780106544495, 0.4165710508823395, 1.1816260814666748, 0.6546476483345032, 0.36207905411720276, -1.9078847169876099, 1.231790542602539, 0.3391273319721222, 1.0999832153320312, 1.3020752668380737, -0.2807355225086212, -0.2959780693054199, 0.4575335383415222, -1.1958822011947632, 0.7165471911430359, -1.325432300567627, -0.18396946787834167, 0.20252689719200134, -0.11657962203025818, -1.400212049484253, -1.506413221359253, 2.0935769081115723, 1.1816813945770264, 0.5552921891212463, -0.22532057762145996, -0.5459766983985901, 0.9640997648239136, -0.45759648084640503, 0.4041130542755127, 0.8062846064567566, -0.661961555480957, -1.8096684217453003, 1.387045979499817, -0.42715445160865784, -0.2820322513580322, -0.8348677158355713, -1.684698462486267, -1.0280345678329468, 1.0259698629379272, 0.9422712922096252, -0.9794763922691345, 0.26586389541625977, 0.8447336554527283, 1.0468199253082275, -0.23385922610759735, -0.7762040495872498, 1.8548836708068848, -0.001610904699191451, 1.3657667636871338, -0.026136677712202072, 1.6601659059524536, -1.1525063514709473, 0.110784150660038, -1.6175427436828613, -0.8435966968536377, -0.35787278413772583, -0.6078000664710999, -0.11971721053123474, -0.14436408877372742, 1.8080672025680542, -0.768817663192749, -0.2437857985496521, 0.8917720317840576, 1.3117607831954956, -1.056397557258606, -0.21471938490867615, -1.2878007888793945, -1.1478900909423828, -0.09158521890640259, 0.8807429075241089, 0.10759372264146805, -0.05185862258076668, -0.16310882568359375, -1.4188826084136963, 0.33735010027885437, -1.1040197610855103, 0.9037536978721619, 0.18449223041534424, -1.246993064880371, -1.0158414840698242, -0.4140991270542145, -1.4899924993515015, 0.41686466336250305, 2.4727425575256348, 0.39501702785491943, -0.25549671053886414, 0.6170067191123962, -1.473385214805603, 2.1733200550079346, 0.6982879638671875, -1.0684787034988403, 0.6656769514083862, 1.519181728363037, -1.0787323713302612, 0.473641961812973, 0.18703581392765045, 0.3161229193210602, 1.7369294166564941, -0.8216834664344788, 1.36632239818573, -0.5171881318092346, -1.250537395477295, 0.35668501257896423, -0.27632081508636475, 0.029226716607809067, -0.9090926647186279, 0.6907795667648315, -0.6223958730697632, -0.54206782579422, -0.22325918078422546, 0.015320472419261932, 0.6374498009681702, 1.0116239786148071, -0.3230643570423126, 0.356499582529068, 1.3146612644195557, -0.8432985544204712, -2.249885320663452, -0.19804924726486206, -1.4359498023986816, -2.1212217807769775, -1.0685194730758667, 2.50421142578125, 0.7013607025146484, -0.66145920753479, -0.7609538435935974, -1.1069684028625488, -0.7940318584442139], [0.4065946638584137, 0.6171542406082153, -2.0283970832824707, 1.2506335973739624, 1.7638858556747437, -1.2518430948257446, -1.1219887733459473, 0.4680090844631195, 1.7783620357513428, 1.5831966400146484, 1.0804067850112915, -0.2589225769042969, -0.5981753468513489, -0.40217894315719604, -0.9467190504074097, -1.6045902967453003, 1.9373294115066528, -0.05943283066153526, 1.2227312326431274, 0.8973038196563721, -0.2971752882003784, 1.2484252452850342, 0.10807035863399506, -0.30515173077583313, 0.8942939639091492, 0.5314021110534668, 1.6027653217315674, -0.7887691855430603, -0.7424569725990295, 0.3558101952075958, 0.8879269957542419, -0.4861353933811188, -2.497466802597046, 0.3180462121963501, 2.0955214500427246, -0.5350421667098999, -1.6323319673538208, -1.0285365581512451, -0.1447240114212036, 0.5275033116340637, -0.6099622249603271, 0.3049418330192566, 0.33063483238220215, -0.9276916980743408, 0.8687241673469543, 1.2362498044967651, -0.976148784160614, 1.3297172784805298, -1.3014744520187378, 0.4163948893547058, -0.29895880818367004, 0.24560396373271942, -1.192744493484497, 0.8815432786941528, 0.5481248497962952, -1.0198428630828857, -0.6291729807853699, -0.8276495933532715, -0.786413311958313, 0.5169771313667297, 0.5206230878829956, 0.47153905034065247, -0.4855846166610718, 0.24723246693611145, -1.6289509534835815, -1.1830220222473145, -1.8581737279891968, 0.5272151827812195, 1.3013100624084473, -0.7867207527160645, 1.5480024814605713, -0.052183136343955994, 1.2262283563613892, 0.37691041827201843, 0.07312940061092377, 0.7732512354850769, 0.8184435367584229, -0.8797794580459595, -0.5762750506401062, -0.14082977175712585, 1.574554204940796, 0.1578560471534729, -1.1880805492401123, 0.6983698010444641, -1.207075834274292, 0.08944835513830185, 0.23038384318351746, 0.41412168741226196, -1.3170990943908691, -1.1816197633743286, 1.1277174949645996, 0.8636512756347656, -0.11212629824876785, -0.5953447818756104, -0.3979322016239166, 2.719053030014038, 1.186305046081543, 0.5459256172180176, -0.9659627676010132, -0.4254215359687805, -0.10740496963262558, -1.8220970630645752, -1.3225330114364624, 0.04490720480680466, -0.592380702495575, 1.5770182609558105, 1.1816432476043701, 0.08156898617744446, 1.4318358898162842, -0.7042598128318787, -0.9157237410545349, -0.6987130641937256, 1.3431487083435059, 0.353674978017807, 1.0383840799331665, -0.35599350929260254, -1.1999534368515015, 0.060645051300525665, -0.18419009447097778, -0.27348268032073975, 1.8165754079818726, -0.6501209139823914, 0.6875178217887878, -0.33564865589141846, -0.7221391797065735, 1.5697945356369019, -0.1699417382478714, 1.474066138267517], [1.3012970685958862, -0.8890240788459778, 0.6835005283355713, 2.8303020000457764, -0.6483827829360962, 0.16606703400611877, 0.39159050583839417, -0.8755418062210083, 0.5349100232124329, 0.14608368277549744, 0.06019379571080208, 0.7796676754951477, 0.13819414377212524, 0.7665284872055054, 0.8514993786811829, 0.17680881917476654, -0.04323658347129822, -1.5048311948776245, -0.9459723234176636, -1.5145063400268555, 0.7848822474479675, -0.5973302125930786, 0.10567965358495712, 0.047784700989723206, -1.1699966192245483, -0.721991240978241, 1.8100742101669312, 0.24766388535499573, 0.23004776239395142, -0.041767507791519165, -0.3272996246814728, 0.5786759853363037, 0.7877863049507141, 0.9294033050537109, 0.10463184863328934, 0.15765038132667542, 1.380251407623291, -0.9888811111450195, 1.0590108633041382, -0.08243648707866669, -1.259773850440979, -1.6241698265075684, 0.08294868469238281, -0.13501860201358795, 0.35312697291374207, -0.3006771504878998, 0.5138839483261108, 1.9855618476867676, 1.3350355625152588, 0.8739362955093384, -0.001192632014863193, 1.2000539302825928, -0.23995789885520935, -0.5990977883338928, -0.29216110706329346, 0.09660319983959198, -0.38024815917015076, -0.9910969734191895, 1.3740732669830322, 2.4338316917419434, 1.6602383852005005, 0.1522144228219986, -0.5168999433517456, -0.10881847143173218, 0.8602548837661743, 0.8478158116340637, 0.39974066615104675, -1.1496282815933228, 0.0639321431517601, 0.825592577457428, -1.601277470588684, 0.965656578540802, -0.26769784092903137, 1.0633667707443237, -0.5730745196342468, 1.0231432914733887, -0.25304552912712097, -0.6254022121429443, 0.6025553345680237, -0.31015297770500183, -0.6998600959777832, -1.3629353046417236, -0.3074309527873993, 1.775692105293274, 0.05505833774805069, -0.39591267704963684, -0.28442543745040894, -0.7466901540756226, 0.19005727767944336, -2.06314754486084, -0.1077672466635704, -0.9096394181251526, -0.3229798376560211, -0.6932346820831299, 1.211177945137024, -0.34867021441459656, 0.14315539598464966, 0.43431946635246277, 1.3381205797195435, -1.9311847686767578, 0.25455403327941895, -1.7230123281478882, -1.313188910484314, -0.8204591274261475, 0.07307598739862442, -1.1831036806106567, -0.30085548758506775, -1.609971046447754, 0.9919850826263428, -0.11219299584627151, 0.3302004933357239, -0.5985848307609558, 0.7907562255859375, -0.7245619297027588, 0.8571275472640991, 1.2583467960357666, -0.4735427796840668, 0.1590729057788849, 0.5442526936531067, -0.16274864971637726, 0.9265170693397522, -0.6689677238464355, 0.034813184291124344, -0.7040562629699707, 0.36766940355300903, -0.0681406557559967, 0.8028731942176819, 0.19728584587574005], [0.5166002511978149, 0.3718350827693939, 0.5176395177841187, 0.11119705438613892, -0.004749227315187454, -1.1676005125045776, 1.360603928565979, -0.4038056433200836, 0.572826623916626, 0.2530311644077301, -0.5654471516609192, -1.499622106552124, 0.3562358617782593, 0.331592321395874, 0.2373192459344864, 0.44809624552726746, -0.8404109477996826, -0.36196282505989075, 0.27285969257354736, -0.11520521342754364, -0.9187480211257935, 0.36821404099464417, -0.8519102334976196, -1.3671026229858398, -0.516308605670929, -1.0433123111724854, 1.2426152229309082, 2.357131004333496, -0.36979326605796814, 0.04558803141117096, -0.03646297752857208, -0.9951309561729431, 0.390930712223053, -0.8019903302192688, 0.886015772819519, -1.1429980993270874, -0.6653619408607483, -0.954255223274231, 0.1768747717142105, -0.18340061604976654, 1.0326870679855347, 1.0460797548294067, 0.5859891772270203, -0.8649829030036926, -0.27247440814971924, -0.959337055683136, -0.760452151298523, 0.07885921746492386, -0.03504712134599686, 0.20234984159469604, 1.3854775428771973, -0.12494593858718872, -0.08066000789403915, -0.27412497997283936, 1.1225221157073975, 1.0569472312927246, 0.7079856395721436, -0.8468422293663025, 1.0863869190216064, 0.8320211172103882, 0.8153210282325745, 0.7014232277870178, 1.514814853668213, -2.2363998889923096, 2.1609065532684326, -0.3078746795654297, -0.9821572303771973, -0.11018890142440796, -0.06890785694122314, -0.24972566962242126, 1.6848200559616089, -2.66766357421875, -0.34749647974967957, 1.5996328592300415, -0.7952759265899658, 1.4165592193603516, -2.3512473106384277, 0.8920450210571289, 1.5482233762741089, -1.8875213861465454, 0.9216681718826294, 2.792330741882324, 0.4758397042751312, 0.6912760734558105, -0.40839454531669617, -1.044775128364563, 0.5751681923866272, -1.334529995918274, 0.23123255372047424, -0.41078871488571167, 0.40440690517425537, -0.0017909164307639003, -0.8906059861183167, -0.1045050173997879, -0.6978714466094971, 1.0154128074645996, 0.1632758229970932, -0.43405014276504517, -0.8757899403572083, 0.3432176113128662, 0.15528512001037598, 1.3319987058639526, -1.1202492713928223, 0.16405019164085388, -0.8400053381919861, -0.9361001253128052, -1.3599554300308228, 0.4941588044166565, -2.135838747024536, -0.02947058156132698, 0.24062038958072662, 1.4476206302642822, -0.2286737859249115, 0.45134204626083374, -0.28658396005630493, -1.2185307741165161, 0.4968215823173523, -0.33734193444252014, 0.17840702831745148, 1.8156355619430542, 1.492779016494751, 0.2617633640766144, 2.1378252506256104, -0.16100306808948517, 0.5877252221107483, -0.423808217048645, -1.4786405563354492, 1.6108839511871338], [-1.117783546447754, 0.2583378255367279, -0.8620555996894836, -0.27807870507240295, -1.3252445459365845, -0.9547833204269409, -1.4190881252288818, 0.6317204833030701, -0.3686351180076599, 1.1173620223999023, 1.3890163898468018, -0.11721374094486237, -0.045996129512786865, 0.7933138608932495, 1.5849779844284058, -0.22262690961360931, -0.757912814617157, 0.372830331325531, 0.8992161750793457, 1.638999342918396, 1.8722319602966309, 0.9694207906723022, 0.23210687935352325, -1.9624967575073242, 0.7664762139320374, 1.572528600692749, 0.8893353939056396, 0.9592294692993164, 1.6999993324279785, -1.5291961431503296, 0.19693216681480408, 0.20241282880306244, 0.9264315962791443, -0.8086308240890503, -0.21088099479675293, 0.4463173747062683, 0.31820183992385864, 0.8507063388824463, 0.7859904170036316, 0.5344428420066833, 0.138225719332695, 1.4636574983596802, 0.03868463262915611, -1.0083527565002441, 0.17603641748428345, -1.8352148532867432, 1.168584942817688, -0.08611323684453964, -1.7018952369689941, -0.698564887046814, -0.12053867429494858, -0.9965546131134033, 0.5010359883308411, 0.8550175428390503, 0.49521133303642273, -1.192097544670105, -0.09845723956823349, -0.4542355239391327, 0.49544528126716614, -0.9874755144119263, -1.1602580547332764, -0.34901565313339233, 0.24515272676944733, 1.3953659534454346, -0.3494727611541748, 0.09475841373205185, 0.6315497756004333, -1.4747263193130493, 0.7554972171783447, 1.1172930002212524, 0.5183646082878113, 0.5822424292564392, 1.1218338012695312, -0.15130199491977692, 1.2866098880767822, -0.23108887672424316, -0.8162621259689331, 1.4057267904281616, -1.278069019317627, 0.6845436692237854, 0.9640765190124512, -0.2875904142856598, -1.755411148071289, 0.2856196463108063, -0.5476337671279907, -0.016195565462112427, 1.2140156030654907, -0.2566925287246704, -1.452238917350769, -0.16855059564113617, -1.24703848361969, 1.4792020320892334, -0.710994303226471, 0.11654072254896164, -1.6311116218566895, -0.9516277313232422, -1.1284867525100708, 1.0764964818954468, -0.4516611695289612, 0.9986801743507385, 0.16090494394302368, -0.20808695256710052, 0.02564949169754982, 0.33557164669036865, -0.8825423717498779, 0.9829254150390625, 0.26650723814964294, 1.768456220626831, 0.940321683883667, 0.2375335842370987, -0.6501938700675964, -1.1208994388580322, 0.08989863097667694, -0.6162654757499695, 0.2726679742336273, -2.6115012168884277, 0.2509326636791229, 0.444291889667511, 0.7216050028800964, -0.6555115580558777, 2.4139537811279297, -2.5495352745056152, 0.0038522661197930574, 0.7423458695411682, -1.1957489252090454, -1.378790259361267, -0.1661055088043213, 2.0808286666870117], [0.26975324749946594, -1.3782140016555786, -1.5780853033065796, 0.7537512183189392, 0.7562263607978821, -0.6172135472297668, -0.033351123332977295, -0.26854997873306274, -0.6629060506820679, -1.6256717443466187, 1.2223899364471436, -0.8816549181938171, 0.5831782221794128, -0.16053487360477448, 0.24458765983581543, 0.1826721876859665, 0.9590024352073669, 0.90067058801651, 0.6570497155189514, -0.2897438406944275, 0.15662138164043427, -1.832996129989624, -1.632547378540039, -1.5367472171783447, -0.8893078565597534, -1.1430630683898926, 0.7563406825065613, 0.5090059638023376, 0.5813795924186707, -1.1990818977355957, -0.04636207967996597, 0.8851603865623474, 0.32352468371391296, -1.4427577257156372, -0.694783627986908, 0.24931250512599945, 1.2002547979354858, -2.2451486587524414, 0.26354894042015076, -0.23323078453540802, -0.037240512669086456, -0.586685836315155, 0.8964551091194153, -0.23936833441257477, 0.41726604104042053, 0.6686733961105347, 0.11193254590034485, -0.14293590188026428, -0.005418695509433746, -1.3060581684112549, 0.8595054149627686, -0.7195802330970764, 0.39530351758003235, -0.27256298065185547, -0.01293297577649355, 0.5490997433662415, -1.626565933227539, 0.3123602569103241, -1.2774578332901, -0.7746238112449646, -0.42035970091819763, 1.1406402587890625, 1.1911680698394775, 0.5326961278915405, -0.9925504326820374, 1.1240569353103638, -0.9870653748512268, 0.14921872317790985, -1.8869688510894775, -1.5446977615356445, 1.730586051940918, -0.9421936273574829, -1.3568581342697144, -1.5776346921920776, -0.222887322306633, -0.17376556992530823, -0.451029509305954, -0.35893186926841736, 0.7147130966186523, -0.5435191988945007, -0.46360278129577637, 0.16243283450603485, -0.3404199779033661, -1.0434914827346802, 0.11778455972671509, -0.09511963278055191, 0.5507141947746277, 0.171688050031662, 1.0110729932785034, 1.6189844608306885, 0.1780090183019638, -0.8539320230484009, -0.16745525598526, 0.540292501449585, 1.6408629417419434, 0.7945625185966492, -0.3684236705303192, -0.20177501440048218, -0.029489992186427116, 0.4293792247772217, 0.1454893946647644, 0.8756510019302368, -0.5479726195335388, 2.000422716140747, -0.3868756592273712, -0.24270936846733093, -0.2933714687824249, -1.0673233270645142, -0.35090330243110657, -1.387742042541504, 0.27679842710494995, -1.049849033355713, -0.3472283184528351, 0.6008750200271606, -0.9468904137611389, -1.8701155185699463, -0.08332003653049469, 0.8555158376693726, 0.6439535617828369, 0.5975571274757385, 0.6374019384384155, 1.4783337116241455, 0.29593825340270996, 0.778015673160553, 0.34663692116737366, -3.0357954502105713, -0.8134716749191284, -1.3131182193756104], [-0.781239926815033, -0.4480955898761749, 0.8756301999092102, 0.06759214401245117, -0.00833248533308506, 0.13325391709804535, 0.05149165168404579, -0.706436276435852, -0.5862107276916504, -0.7072077393531799, -0.7627639770507812, 1.1757054328918457, -1.5674501657485962, -1.7549195289611816, -1.343733787536621, -1.492558240890503, 0.9218448996543884, -0.5166969895362854, 2.099148988723755, -0.26364365220069885, -0.41201475262641907, -0.5965595245361328, -0.06204599142074585, 1.115903615951538, -2.269744396209717, 0.720170795917511, 1.0457727909088135, -1.0958666801452637, 0.523259162902832, 0.06416439265012741, -0.6917681097984314, 0.2614745795726776, 0.5986984372138977, 1.2060130834579468, -1.4465311765670776, -0.21665380895137787, 0.9392386674880981, -0.9418367147445679, 0.7422623038291931, 0.6994324922561646, -0.9482300877571106, 1.042739987373352, -3.457988977432251, 0.691627025604248, -0.8423312902450562, -0.4852445721626282, -0.28437191247940063, 0.33608880639076233, -0.6852841377258301, 0.6197010278701782, 0.8272746801376343, -0.03927009552717209, -0.2814134955406189, -1.9382643699645996, -0.029087290167808533, -0.9384210109710693, -0.8756120204925537, -1.6371015310287476, -0.5905622243881226, 0.027955187484622, -0.4701286256313324, 1.794097661972046, 1.628124475479126, 0.08666250109672546, -0.5102596879005432, -1.4698251485824585, 0.011968446895480156, 0.21195025742053986, -1.343667984008789, 0.4760398864746094, 0.3433624505996704, 0.4779590964317322, 0.3673486113548279, -1.9347695112228394, 1.7444374561309814, 1.1323024034500122, -1.037310242652893, -0.43029963970184326, -0.31810665130615234, 1.0374356508255005, -0.8930041193962097, -1.0436605215072632, -0.3211021423339844, 0.6518227458000183, 0.9257311820983887, -0.4940741956233978, 1.4687016010284424, 0.16704526543617249, 0.02523861639201641, 0.7431730628013611, 0.2098592072725296, -0.691521942615509, -0.7875882387161255, 0.022149881348013878, -0.24717780947685242, 1.6555497646331787, 1.8341660499572754, 0.06086762621998787, -1.9766634702682495, -1.4026517868041992, 0.1464761644601822, 0.053265318274497986, -1.9221206903457642, 0.038296692073345184, 0.20236243307590485, -1.7692099809646606, -1.5850701332092285, -1.1805850267410278, 0.09069585055112839, -0.3204841613769531, 0.4247019290924072, 0.8366325497627258, 0.11556188017129898, -1.1972641944885254, -0.9985888600349426, 1.8526031970977783, 1.3200496435165405, -0.5346185564994812, -1.5409576892852783, 0.41377368569374084, 1.2193537950515747, 0.3464807868003845, -0.36850082874298096, -0.37950488924980164, 2.0961344242095947, -0.6981217861175537, -1.3669148683547974, 0.3451535403728485], [-0.0426984541118145, 0.4695527255535126, 0.49639254808425903, -0.6126411557197571, 0.897931694984436, -1.0556124448776245, -0.5808061957359314, -1.2073365449905396, -1.439803957939148, 0.7000480890274048, -1.4850678443908691, 1.1889311075210571, 0.46832799911499023, 0.34551501274108887, -1.112715244293213, -0.2242773324251175, -0.17864039540290833, 0.3783039450645447, 1.0381596088409424, -2.1573946475982666, -0.9990082383155823, 0.25373077392578125, 0.3917118012905121, -1.0313564538955688, 0.666218101978302, -0.08480941504240036, -0.22236758470535278, 0.363966166973114, -0.5612441897392273, -0.6057363748550415, -1.375095009803772, 1.5691742897033691, 0.3159302771091461, -1.2808934450149536, -0.9720355868339539, 2.0185344219207764, 0.33609044551849365, 0.2614986300468445, -0.8470391631126404, 0.6867311596870422, 0.7709226012229919, -1.1867233514785767, -0.3950383961200714, -0.7017769813537598, 0.7625503540039062, -0.11483205109834671, -0.19437631964683533, 0.13993556797504425, 0.6031279563903809, -1.0499061346054077, 1.5378801822662354, -0.16907289624214172, -0.14758175611495972, -1.458484172821045, -1.6633930206298828, 0.20768921077251434, 0.2661805748939514, 0.47044631838798523, -2.2246921062469482, 0.24490006268024445, -0.9594683647155762, 0.7904053926467896, -1.8115897178649902, 0.6413095593452454, 0.19011452794075012, 0.47639116644859314, 0.9507083892822266, 0.11843665689229965, 1.6924101114273071, -1.22482430934906, -0.29754212498664856, 0.4483836591243744, 0.49716928601264954, -0.8474916219711304, -1.0676524639129639, -0.38986754417419434, -0.19173140823841095, 1.4645262956619263, 0.5189773440361023, 1.3859483003616333, -0.7158039212226868, -0.4919881522655487, 0.26580071449279785, -0.418558269739151, 0.22932758927345276, -0.7989467978477478, -0.2742254436016083, 0.6920485496520996, -1.943447232246399, -0.9240918159484863, 0.2922920286655426, -0.6965076327323914, -0.7507445812225342, -0.5524939298629761, 0.06272586435079575, 1.075475811958313, 1.418715000152588, -0.16410017013549805, 0.45131587982177734, 1.46134352684021, 1.3139979839324951, -0.8449932336807251, 1.5204799175262451, -1.5551916360855103, 0.8719222545623779, 1.5940454006195068, -0.8540745377540588, 0.5090672373771667, 1.065971851348877, 1.3992787599563599, 0.394603967666626, 0.34209775924682617, 0.550442099571228, 0.8595373630523682, -1.193722128868103, -0.8405243158340454, 0.7707321643829346, 2.4078657627105713, 0.5412307977676392, -1.2819668054580688, -0.24646662175655365, -0.3893234133720398, 1.9807778596878052, 0.25440651178359985, -0.12503454089164734, -1.7238187789916992, -0.32837674021720886, 0.9219086170196533], [-0.17425812780857086, -0.48879584670066833, -0.38473114371299744, -0.9204152822494507, 1.9380922317504883, -0.31533291935920715, 1.3701194524765015, -0.1368340402841568, 1.1142245531082153, 1.6463171243667603, -0.7282148003578186, -0.29442456364631653, -0.3425503671169281, -1.895259141921997, -0.04464724287390709, 1.0898911952972412, -0.2584564685821533, -0.7773514986038208, -0.5583814382553101, -1.2044278383255005, -0.025121888145804405, -0.2621970772743225, 0.34923291206359863, -0.1218113973736763, -0.04031406342983246, 0.5251116156578064, -1.2372663021087646, 0.30130165815353394, 1.685643196105957, 0.10138348489999771, 1.3455591201782227, -0.9954193234443665, 1.147020697593689, -1.1254620552062988, -0.9495582580566406, -1.1480058431625366, 0.5254939198493958, -1.1841274499893188, 0.18573445081710815, 2.2239367961883545, -0.2841190695762634, -1.6474288702011108, -0.10725840926170349, -0.24807575345039368, -0.5507386326789856, 0.7601838707923889, 0.8413646817207336, 0.3068012595176697, 0.337480366230011, -1.0864869356155396, 0.18785999715328217, 0.7041370868682861, 0.5728713274002075, -0.26898130774497986, 0.5395395159721375, 1.5256600379943848, -0.14158281683921814, -0.0302211232483387, -0.3942800760269165, -0.6112408638000488, 1.122680425643921, 0.7794691324234009, 0.10523494333028793, 1.5492743253707886, 0.6690903902053833, 0.981644332408905, -1.616376280784607, -1.1565359830856323, -0.1571090817451477, 1.0150226354599, -0.3733052909374237, 0.4104527235031128, 0.1386667788028717, -0.14221897721290588, -1.5506104230880737, -1.4159198999404907, -0.20441210269927979, -0.15679748356342316, 0.12954339385032654, 0.1343768686056137, 0.036372844129800797, -0.4987471401691437, -0.15998950600624084, 0.3153054416179657, 0.5232757329940796, 1.227549433708191, 0.6472606062889099, 1.2085622549057007, 1.0747193098068237, -0.11423817276954651, 0.5188882350921631, 0.7129391431808472, -0.8860849738121033, 1.090954303741455, 0.346873015165329, 0.7392697930335999, 0.4809889495372772, 0.8232365250587463, 1.2400085926055908, 0.0848560631275177, -0.5378279089927673, -0.09370596706867218, 0.12947174906730652, 0.25335168838500977, 0.6176750063896179, -0.7887099385261536, 0.9570348262786865, -1.8273906707763672, -1.273972749710083, 0.9962722063064575, -0.6434611678123474, 1.6990920305252075, 0.0047663431614637375, -0.13096261024475098, -0.8248211741447449, -0.205110102891922, 0.3570122718811035, 0.26134157180786133, -0.2717079222202301, 1.3950282335281372, -0.19216513633728027, 0.632013738155365, -0.3375694453716278, 0.7108540534973145, -0.7930124402046204, 0.41373714804649353, -0.6139930486679077, -0.09225893765687943], [-0.05411228537559509, -2.243133544921875, -0.6891900300979614, -0.50473552942276, 0.3267310559749603, 0.03887812793254852, 0.44664523005485535, 1.2734378576278687, -1.9207935333251953, -0.7133700847625732, 0.6074567437171936, 0.01352581474930048, -0.739717423915863, 0.6406126618385315, -0.048774074763059616, 0.6497613787651062, 0.30543774366378784, -0.526313304901123, -1.2597719430923462, 0.6584674119949341, -1.5501432418823242, -0.346099853515625, -0.25892776250839233, 0.7292901277542114, 0.5480188131332397, 0.35936376452445984, -2.059908390045166, -1.9320746660232544, -0.057053904980421066, 1.4714869260787964, 2.369520902633667, 0.9167271256446838, 1.5548278093338013, -0.07900483161211014, 1.0586050748825073, 1.1214137077331543, 0.8418004512786865, -0.24046781659126282, 0.14641210436820984, 0.34646740555763245, 1.3252671957015991, 1.0744277238845825, 0.7978953123092651, 1.1806139945983887, -1.3746238946914673, -0.5997911095619202, -1.8605538606643677, -2.5534040927886963, 0.06378492712974548, -0.6331507563591003, -1.388122797012329, 0.196604385972023, 2.183788776397705, -0.8230065107345581, -0.017527567222714424, 1.5794743299484253, 0.46399614214897156, -0.5743453502655029, 0.36614012718200684, 0.2033238559961319, -0.1879357099533081, 0.1204429492354393, -1.4574836492538452, -0.2709541320800781, 0.42631304264068604, -0.6105365753173828, -0.64340740442276, -1.1480125188827515, 0.017676569521427155, -0.011315904557704926, -0.42798173427581787, 0.795068085193634, 1.5721662044525146, 0.08293025195598602, -0.42060530185699463, 0.2651497423648834, -0.8181062936782837, -1.302964448928833, -0.6515328288078308, 0.47713392972946167, -0.6002343893051147, 0.20606230199337006, 1.4941151142120361, 1.6384271383285522, 0.3874424695968628, -1.162537693977356, 0.1676367074251175, -1.107696533203125, -0.7519698739051819, -1.452393889427185, -1.6612497568130493, -0.24492159485816956, -0.35450276732444763, 0.2394016832113266, -0.29722660779953003, 0.9024902582168579, -1.3278748989105225, -0.24200554192066193, 0.195702463388443, 0.36913391947746277, 0.5723692774772644, -0.4015365540981293, -1.5704203844070435, 0.5329243540763855, -0.942370593547821, -1.5245076417922974, 1.3326084613800049, 0.9101752042770386, -0.9055973291397095, -0.8009964227676392, -0.3590620160102844, 0.3359496295452118, 1.1440691947937012, -0.11806249618530273, -1.207943320274353, 1.5923572778701782, 1.1296029090881348, 1.124068260192871, 1.6797047853469849, 0.950372576713562, -1.892655372619629, -1.4339921474456787, -0.3639986515045166, 0.7674978375434875, -0.008729065768420696, 0.6406177282333374, 1.1585676670074463, -0.8018815517425537], [0.2093820422887802, 0.6888322830200195, 1.902999997138977, -1.0796351432800293, 0.2618268132209778, -0.2271520495414734, -0.5819028615951538, -0.6719762682914734, -0.03203534707427025, 0.7370111346244812, 1.0214083194732666, 1.17190682888031, -1.0216954946517944, -0.4466184973716736, 2.2637851238250732, -1.0200445652008057, 0.0626915767788887, -1.4972307682037354, 0.657680094242096, -1.7965035438537598, 0.8799716830253601, 1.6583689451217651, 1.1055222749710083, 0.3552260398864746, 0.5957100987434387, 1.5208064317703247, -0.31233808398246765, -0.013904714956879616, -1.4197791814804077, -0.7646453976631165, 0.35718122124671936, -0.4833788275718689, -1.3783047199249268, 1.0329596996307373, -0.8963655233383179, -0.6325464248657227, -0.8403244614601135, -1.619989275932312, 2.4450314044952393, -2.342562198638916, -0.0768815129995346, -0.10563269257545471, 0.08463292568922043, 0.6041708588600159, 1.8209470510482788, 0.46501925587654114, 2.078653573989868, -1.83354651927948, -0.46489471197128296, -2.3881583213806152, -0.5627370476722717, 0.8032186627388, -0.48653966188430786, 1.3713105916976929, 1.693616509437561, -1.3482900857925415, 1.9799710512161255, 0.23565547168254852, 0.44726860523223877, 1.0001646280288696, 0.20961794257164001, -1.1845327615737915, 0.529313325881958, -0.04265187308192253, 0.12350817024707794, 0.06595982611179352, 0.6241220831871033, 0.10559368133544922, 0.275540828704834, 0.5369042158126831, 0.5717282891273499, -1.1914372444152832, -0.7638515830039978, 0.4608525335788727, -0.4162355661392212, -0.5430403351783752, -1.5529475212097168, -1.560945749282837, -0.7013238072395325, 0.8232101798057556, -0.16832397878170013, 0.6840124726295471, 0.4785202741622925, 0.9130313396453857, 1.4313483238220215, -0.1788894236087799, -0.9373488426208496, 0.18567724525928497, -0.7681679725646973, 0.20656169950962067, 0.1942722052335739, 0.8386368751525879, -0.28409165143966675, -1.625649094581604, 0.9375629425048828, -0.6434534192085266, 0.8083195686340332, 0.7614936232566833, 0.20645484328269958, -0.2866832911968231, 2.1432313919067383, 1.7201752662658691, 0.2018708884716034, 1.558354139328003, 0.2939775884151459, -0.330331027507782, -0.23385775089263916, -2.404305934906006, 1.8050813674926758, 0.7896145582199097, -0.34344643354415894, -0.5514686703681946, 0.6239396929740906, 1.563602089881897, -0.12970732152462006, 1.8553717136383057, 0.07037176936864853, -0.06489024311304092, 1.2429509162902832, -0.5946885347366333, 0.5655447840690613, -0.440897673368454, -0.2866838276386261, -1.4321703910827637, -0.9114280343055725, 0.46109598875045776, -1.4903357028961182, 0.6196268200874329], [-0.6246849894523621, -2.9561009407043457, -1.728348731994629, -1.055881381034851, 0.5256050825119019, -0.04394958168268204, -2.039875030517578, 0.6231623888015747, -0.6899824142456055, 0.11910071969032288, -0.6925210952758789, 0.0653863474726677, 0.03838175907731056, 1.0071439743041992, -0.5611307621002197, -0.9854578971862793, -0.5325320959091187, -0.22091837227344513, 0.4200741648674011, -1.4400897026062012, -0.7933651208877563, 1.2535725831985474, -0.21166229248046875, -0.2410253882408142, -0.4213049113750458, -0.6866877675056458, -0.7718992829322815, 0.4627397656440735, 0.8193471431732178, 0.11556816846132278, 0.3453054428100586, -1.1486141681671143, 0.5492669343948364, -0.6209805011749268, 0.7564553022384644, 0.41161492466926575, 1.0750495195388794, 0.530354380607605, 1.129598617553711, -1.6673208475112915, -0.13074542582035065, -0.9519011974334717, -0.7518622875213623, 0.9292863607406616, -1.2774685621261597, -0.1657436639070511, -0.5061001777648926, -1.6072920560836792, 1.1302989721298218, -1.0047744512557983, -0.6502974033355713, 1.1156712770462036, 0.19656376540660858, -0.3446739912033081, -1.54036283493042, 0.17518967390060425, 0.7635700702667236, -1.077826976776123, -0.528290331363678, -1.8425934314727783, -1.5188368558883667, 0.5747544765472412, 0.1596153825521469, 0.6325332522392273, 0.5009214878082275, -2.0589663982391357, 2.0964207649230957, 0.14104455709457397, 1.3787132501602173, 0.6728799939155579, -0.3246528208255768, 1.145310878753662, 0.40715304017066956, -2.6863605976104736, 0.4077479839324951, 1.3234246969223022, -2.0562222003936768, 0.12417405843734741, -1.1974694728851318, -0.021195026114583015, 0.04377591237425804, 0.631206750869751, 0.0597558468580246, -1.0738601684570312, -0.21384519338607788, -0.3066475987434387, -2.295720338821411, -1.8760408163070679, 0.10285986959934235, -1.0075697898864746, -0.11489663273096085, -1.4588104486465454, 0.3957195580005646, -1.0692113637924194, -0.5497905611991882, -0.9636638760566711, -0.8290184140205383, -0.16964152455329895, -0.058924175798892975, -0.6295578479766846, -0.2762806713581085, -1.0305482149124146, 0.28473666310310364, 1.4533462524414062, 1.6215651035308838, -0.2702754735946655, -1.2320137023925781, 0.015329006128013134, 0.3516295552253723, 0.19864781200885773, 0.1776529848575592, 1.2511669397354126, 0.20887631177902222, 0.6698211431503296, 1.4122354984283447, -1.4780230522155762, 0.195454403758049, 0.036726828664541245, 1.3360353708267212, 1.5405033826828003, -1.6206923723220825, -1.1007837057113647, -0.03724588826298714, 2.4732751846313477, 2.019787549972534, -0.6770146489143372, -2.477794885635376, -1.4369322061538696], [0.2971450984477997, -0.6761437654495239, -0.71974778175354, -1.2686474323272705, 0.43261706829071045, -1.248489260673523, 0.6781628727912903, 1.2053172588348389, -1.734352946281433, 0.7438629865646362, -0.9681196808815002, -1.0611693859100342, 0.8336210250854492, -0.2702485918998718, 0.5372074842453003, -0.6370469927787781, 2.2076761722564697, 1.1347063779830933, -0.49242928624153137, -0.14978015422821045, -0.5517497062683105, 1.157293677330017, 0.19114230573177338, -1.3365601301193237, 1.275719404220581, -0.028001047670841217, 1.7887974977493286, 0.45320630073547363, -1.4211459159851074, -3.156428575515747, -0.0618792399764061, -0.2763940095901489, 1.8080534934997559, 0.2631760835647583, -0.5016417503356934, -0.6202940344810486, -1.8429350852966309, 1.1313910484313965, -0.41060060262680054, 1.2316062450408936, -0.886177659034729, 0.5651801228523254, 0.12669648230075836, 1.0567480325698853, 0.18687976896762848, -0.04836627468466759, -1.683240294456482, -0.019161760807037354, 0.8690993189811707, -1.0644837617874146, 1.4725676774978638, 0.6003062129020691, -0.3944469094276428, -0.5611193180084229, 2.69380784034729, -1.1129604578018188, 0.5760061144828796, -0.21736931800842285, -0.04403584450483322, -0.3842185437679291, 0.07822314649820328, -0.4496462941169739, -3.1049413681030273, 0.3772514760494232, 0.5362486243247986, -1.7843754291534424, -0.2544083595275879, 0.33974820375442505, -0.9455527067184448, 1.0785192251205444, -1.596695065498352, 1.5384275913238525, -0.35174256563186646, 0.023654790595173836, -0.6328115463256836, -0.6613070368766785, -0.4765620827674866, -1.4465128183364868, -0.07262590527534485, 0.24138016998767853, -0.8719232082366943, 0.5331794619560242, -0.015630658715963364, -0.8987240195274353, -0.5490732789039612, 2.0254061222076416, -0.8500559329986572, 0.14076527953147888, -1.4734807014465332, 1.324480414390564, 0.5210182666778564, 0.31360793113708496, -2.265977621078491, 0.5465618371963501, 0.05764389410614967, -0.9307091236114502, -1.1140369176864624, 0.2012656182050705, 0.4200170338153839, 1.1060835123062134, -0.5557715892791748, -0.6441503763198853, -0.14725510776042938, 0.14387445151805878, -0.6491245627403259, 0.2046809047460556, -0.4202827215194702, 0.03406607359647751, 2.0493836402893066, 1.1838444471359253, -1.1365069150924683, 1.8499622344970703, -0.06888902932405472, 0.6184632182121277, 1.562589168548584, -1.7513444423675537, 0.5281266570091248, -1.0168882608413696, -0.3640812337398529, -0.2400483936071396, -0.9694293141365051, -0.9001749157905579, 0.30362144112586975, 0.03261013329029083, -0.25934740900993347, -0.0361735038459301, 0.28001531958580017, -0.6932361721992493], [-0.21409012377262115, -0.3730124235153198, -2.1535651683807373, -0.05273216590285301, -1.3334980010986328, 2.4103896617889404, 0.09076917916536331, -1.3720452785491943, -0.007265046704560518, 1.3483206033706665, -0.466360479593277, -0.753539502620697, -0.14881090819835663, -0.22713343799114227, 1.6146844625473022, 0.5230532884597778, -1.5686538219451904, -0.30685457587242126, -1.1056839227676392, 0.2605447471141815, -0.5903225541114807, -0.271584689617157, 1.3062666654586792, 0.6876598596572876, -1.0327377319335938, -0.5656210780143738, 0.3238675892353058, 1.050360918045044, -0.3556990921497345, -0.24032022058963776, -0.6541116833686829, -0.379138320684433, 1.183627724647522, -0.2936241328716278, 1.3433375358581543, 0.48367390036582947, -1.231766939163208, 1.0142409801483154, 1.1622565984725952, -1.3627394437789917, -1.4001094102859497, -1.343499779701233, 0.9563096761703491, -0.3787091076374054, 2.2553794384002686, -0.44571611285209656, -1.0928115844726562, 0.4348619282245636, 0.891187310218811, 0.3111162781715393, 0.5366272926330566, -0.801254153251648, -1.3853824138641357, -0.18164493143558502, -0.17496436834335327, 1.7882301807403564, 2.010002613067627, 0.9597629904747009, -0.8732207417488098, -1.218634843826294, -1.9974274635314941, 0.047593604773283005, 0.10494158416986465, 0.917474627494812, -0.9736865162849426, 1.125768780708313, 0.6031918525695801, -1.5337899923324585, -2.0987906455993652, 0.566375732421875, -0.7965033650398254, -0.7094354629516602, 1.1417204141616821, 0.5990583896636963, 1.2647030353546143, 1.0152909755706787, -0.5317756533622742, 1.5157833099365234, 1.2634022235870361, 0.33292385935783386, -1.9173325300216675, 1.3455595970153809, -1.005810260772705, -0.003988124430179596, 0.6733497977256775, -0.8979270458221436, 0.023176567628979683, -0.9656083583831787, 0.12543341517448425, -0.7024132609367371, 1.1016895771026611, 0.4571228623390198, -2.409026861190796, 0.6761590242385864, -1.5568112134933472, -1.831783652305603, 0.8890433311462402, -1.0200316905975342, -0.4143337905406952, -0.45379137992858887, 0.18773877620697021, 0.06757992506027222, 0.27704113721847534, -0.4087955355644226, 0.766008734703064, 0.5910008549690247, -0.013764988631010056, 1.838842749595642, 0.04407487064599991, 1.348944067955017, 0.5624471306800842, -1.1682049036026, -2.037888765335083, -0.4072304666042328, -0.7809930443763733, -0.9305076599121094, 0.7028926014900208, 1.311948299407959, 0.8092638254165649, -0.9295450448989868, 0.3262002170085907, 1.9494057893753052, -0.3768270015716553, -2.386042594909668, 0.6537052989006042, -0.991737961769104, -2.0391156673431396, 0.4598441421985626], [-0.7973616719245911, -0.9278777837753296, 0.6491450667381287, 0.10501212626695633, 0.2814001441001892, -0.6149962544441223, 0.6747648119926453, 1.8364278078079224, -1.1035728454589844, 0.8071211576461792, 1.1589233875274658, 0.1306152492761612, -0.06433935463428497, 1.2349498271942139, -0.3335024416446686, 0.37555742263793945, 1.1791324615478516, 1.5968660116195679, 0.8342259526252747, 0.256600022315979, 0.834682047367096, -0.2534593939781189, -0.11931265145540237, 0.3284345269203186, 0.5801201462745667, 0.7409899830818176, -2.2130391597747803, -2.766707181930542, 0.11463487893342972, -0.25056397914886475, 0.21766217052936554, 0.9100053906440735, 0.34461650252342224, 1.1456730365753174, -0.7174646258354187, -0.5732777714729309, 1.422972559928894, -0.24739409983158112, -0.3831256628036499, -0.39136484265327454, 1.2956724166870117, -0.8549808859825134, -0.6557433605194092, 1.2669719457626343, 1.2696583271026611, -0.11940848082304001, 0.6288303732872009, 1.7850197553634644, 1.9082897901535034, 0.45526009798049927, -0.3256683051586151, -1.3836703300476074, -0.7988153100013733, -0.732756495475769, -1.8559606075286865, -0.20320755243301392, -0.32344362139701843, 0.07996252924203873, -1.4395147562026978, 0.9962015151977539, 1.074485182762146, 0.8293875455856323, -1.84060537815094, 0.42156779766082764, -1.364363193511963, -0.5913557410240173, 0.024235043674707413, 1.495041847229004, -1.338923454284668, -0.22490739822387695, -0.08369883894920349, -1.0464978218078613, 1.461958408355713, 1.0691955089569092, 0.8725233674049377, 0.7501778602600098, -0.4298323094844818, -0.16375978291034698, -0.7996968030929565, -0.8567664623260498, 0.5402581691741943, 0.6519982814788818, -1.089150071144104, -2.4246764183044434, 0.45277974009513855, -0.15660136938095093, 0.6381690502166748, -0.32738956809043884, 0.1270274519920349, -0.16119050979614258, 0.33685559034347534, -0.15025652945041656, 0.46216800808906555, -1.1374194622039795, -0.14927224814891815, 1.4588849544525146, 0.8993914127349854, -1.6126365661621094, -0.360182523727417, -0.3610803186893463, -2.18917179107666, 2.8925631046295166, -0.7189017534255981, -2.532945394515991, -0.20522131025791168, -0.47143417596817017, -1.3691949844360352, -0.12257906794548035, -0.9627601504325867, -1.9123613834381104, -0.5867320895195007, -0.24983417987823486, 1.0318821668624878, 0.8735132217407227, 0.5818920135498047, 1.7296737432479858, -0.327400267124176, 1.0717480182647705, -1.3740336894989014, 1.6985162496566772, 1.8611111640930176, 1.038332462310791, -0.3003861904144287, 0.7447733283042908, -0.17790938913822174, -0.878118634223938, 0.05661074444651604, -1.1402032375335693], [0.21365080773830414, -0.9363885521888733, 0.06864796578884125, 0.2081851065158844, -0.2911786735057831, -0.4088402986526489, -0.7608641982078552, 0.32538628578186035, -0.04769878461956978, -0.38323938846588135, -0.1366586536169052, 2.097231388092041, 0.1985437422990799, 1.0108004808425903, -1.1435086727142334, -0.8261528611183167, -0.9259807467460632, -0.3708863854408264, 2.206575393676758, -0.06137194111943245, 0.9159995913505554, -0.8298834562301636, -0.86086106300354, -1.2760547399520874, 0.1562584936618805, -0.9893848299980164, 0.5132121443748474, -0.021320391446352005, 0.21278850734233856, -0.058377135545015335, 0.3113943636417389, -1.0587397813796997, 0.1541525423526764, 0.38862425088882446, -0.3110160827636719, 0.4203874170780182, -0.21522822976112366, -0.690232515335083, 0.7283750772476196, 0.8299908638000488, -1.1944242715835571, -0.9748319983482361, 1.3471393585205078, -2.407397508621216, -0.5665455460548401, -0.040179505944252014, 0.3516104221343994, -0.38603490591049194, -1.5251175165176392, 0.19078484177589417, -0.9120528101921082, 0.5514050722122192, 0.6715399026870728, -0.40699687600135803, 0.3170638382434845, 0.10151305049657822, 0.5251103043556213, -1.109735131263733, 0.6247642636299133, -0.7401387691497803, 1.4773261547088623, 0.5075405240058899, 1.2386220693588257, 0.035725489258766174, 0.03812336549162865, 1.139603614807129, 0.1649179458618164, -1.304731011390686, 0.12245075404644012, 0.21569418907165527, -1.0741817951202393, -1.7483644485473633, -0.3753942847251892, -0.7416713833808899, 0.7666995525360107, -0.7629594802856445, 1.242687463760376, 0.299618124961853, -0.3643641471862793, 1.038059115409851, -1.0982286930084229, 1.7350306510925293, 1.20091712474823, -1.4177298545837402, 1.3869770765304565, -1.3161425590515137, -0.8579936027526855, -0.12188854813575745, -1.1510672569274902, 1.2407938241958618, 0.295081228017807, 0.7755509614944458, 2.6496269702911377, 0.9155728220939636, 1.112184762954712, 0.18503719568252563, -0.6971719861030579, 0.6389463543891907, -1.3912497758865356, 0.8247156739234924, 0.07857012003660202, 0.26866739988327026, 0.2117946296930313, -0.4685438871383667, -1.2957944869995117, -1.5274944305419922, 0.9128754138946533, 0.5527074337005615, -0.16204623878002167, -1.089636206626892, -0.4859040677547455, 1.502202033996582, 0.4075164198875427, -0.6961768269538879, 0.5207324028015137, -0.8532535433769226, 0.6366468071937561, -0.7314636707305908, -0.37062889337539673, 1.1783647537231445, 0.8298256993293762, -1.369714379310608, 1.2636798620224, -0.1635832041501999, 0.2979133427143097, -0.2589019536972046, 1.194420337677002, 0.03417646512389183], [-0.5065619349479675, -0.3024793863296509, 0.8520359992980957, -2.6328587532043457, -0.0646398738026619, -1.1308053731918335, 1.1581556797027588, 0.9154115319252014, -1.3465394973754883, -1.5494322776794434, -1.2263929843902588, -1.1303558349609375, -0.5768336653709412, -0.06176992133259773, 0.679193913936615, -0.31589475274086, -0.4750059247016907, -1.311238408088684, 0.8411339521408081, -0.4057596027851105, 1.243553638458252, 0.36837267875671387, 0.530860185623169, -1.8810919523239136, 0.17424975335597992, 0.011388067156076431, 0.360578715801239, -1.2686147689819336, 0.18587726354599, 0.5736208558082581, -0.06268496811389923, -1.5470799207687378, 0.6820282340049744, 0.8209022879600525, 1.5883815288543701, -0.8431447148323059, -0.41063785552978516, -1.3623102903366089, -0.36277303099632263, -0.9412590265274048, -0.688189685344696, 0.4455285370349884, 1.679552674293518, 0.774949848651886, 0.0020421885419636965, 0.5651196241378784, -0.010890768840909004, 0.1166715919971466, -1.7419370412826538, 0.8382942080497742, 0.19878023862838745, 0.2974987328052521, 1.4160826206207275, 0.8163318037986755, 1.8069791793823242, -0.9464813470840454, 0.5000841617584229, 0.39032191038131714, 0.9503348469734192, 0.746442973613739, -1.1609796285629272, 1.1253917217254639, -0.8400513529777527, -1.1763707399368286, -0.8311179876327515, 1.6011167764663696, 0.6085337400436401, 0.36046919226646423, 2.3188953399658203, -1.282637119293213, 0.5407384037971497, 0.9745416641235352, -0.5086112022399902, 0.06772937625646591, -1.6761529445648193, 0.48400381207466125, -0.7258495092391968, 1.2315312623977661, 2.1996243000030518, 0.06106723099946976, 0.6527523398399353, 0.45394167304039, 0.9620591998100281, 1.7324529886245728, -1.3974641561508179, -0.28916218876838684, -0.9267811179161072, -0.1399228572845459, -0.2842878997325897, -1.2837097644805908, -1.0959324836730957, -0.19702030718326569, 0.9571974277496338, 1.0653769969940186, -0.882905125617981, -0.26685693860054016, 0.39626702666282654, 0.5806485414505005, -0.3394523859024048, -1.191702127456665, -1.727617859840393, 1.6093348264694214, -0.3547222316265106, -0.9144349098205566, -0.05345665663480759, -0.9156510233879089, -0.5940700173377991, 0.31149017810821533, 0.21315225958824158, -0.9419509172439575, 1.2140417098999023, -1.277095079421997, -0.3761197626590729, -0.8638750910758972, 0.043566592037677765, -0.24331143498420715, 0.3832482099533081, 1.1529515981674194, 1.6465741395950317, 0.30000221729278564, 0.8026558756828308, -0.015680687502026558, -0.04103328660130501, 1.5891313552856445, 0.22810710966587067, -0.8549426794052124, 0.4825262129306793, 1.329406976699829], [1.027220606803894, 1.7301228046417236, -0.34559834003448486, -0.2633266746997833, -1.0109480619430542, 1.0994815826416016, 0.06319981813430786, 1.8494668006896973, 1.3274673223495483, 0.21385866403579712, -0.2286866307258606, -1.0304542779922485, 2.6107916831970215, -0.238671213388443, 1.2195554971694946, -0.8297210335731506, 0.8959353566169739, 0.8120238780975342, -0.49674639105796814, 0.9630206227302551, 1.1999719142913818, -0.3674612045288086, 0.2166913002729416, -0.2631010115146637, -0.8829414248466492, -1.5680220127105713, 0.1467902958393097, 1.474554181098938, -1.5819683074951172, 0.7721244096755981, -0.03886409103870392, 1.5074259042739868, 0.3365227282047272, -0.9073224067687988, 0.7447463870048523, -1.577531337738037, 0.5864871144294739, 0.05720837041735649, -1.4978028535842896, -1.4662846326828003, 0.8892354965209961, 0.05315522104501724, -2.1327388286590576, -1.0944287776947021, 0.4271831512451172, -0.09923111647367477, 0.9236114025115967, 0.1257735639810562, 1.11435866355896, -1.4783391952514648, -1.1207257509231567, 1.2599996328353882, -1.2245622873306274, -1.166947364807129, 0.5628848671913147, -0.18203628063201904, 1.3150194883346558, -0.26072150468826294, 1.5170165300369263, -0.7327746152877808, -0.967338502407074, 0.4431706666946411, -1.43757164478302, 1.065110683441162, 1.3566519021987915, -0.5980780124664307, -0.6603139638900757, -1.0932857990264893, -1.4373503923416138, 0.16826112568378448, 0.41521474719047546, 0.8971379399299622, 0.42097046971321106, -0.07339683920145035, 0.3464873433113098, 0.1651284098625183, -0.06762846559286118, 0.3322174847126007, 0.35097160935401917, -1.1772382259368896, 0.1518407016992569, 3.424546241760254, 0.7619858980178833, 0.8204770088195801, 0.14973171055316925, -0.09592833369970322, -0.2599170506000519, -0.6255000829696655, 0.24317488074302673, -0.03840929642319679, -1.0943446159362793, -0.7980312705039978, 1.8409916162490845, -0.02133948542177677, -1.3991968631744385, 1.7242364883422852, 0.6616572141647339, -0.0771380290389061, -0.12387022376060486, -2.1975154876708984, 0.8155677914619446, -0.9282603859901428, -2.1072633266448975, -2.3488776683807373, 0.019529668614268303, -1.1277917623519897, -0.938786506652832, -0.2852833569049835, -1.1893606185913086, 0.7604009509086609, -1.0781595706939697, -0.550190806388855, -0.44935664534568787, 0.28374916315078735, 0.15553666651248932, 1.165260672569275, 0.9806491732597351, 0.048026036471128464, 0.40407687425613403, 0.48078039288520813, 0.20557627081871033, 1.1622059345245361, -0.1635674685239792, -1.1979674100875854, -1.1521564722061157, -2.0382814407348633, -0.6906809210777283, 2.3334527015686035], [1.684230923652649, -0.5876181125640869, 0.7631697654724121, 1.0785142183303833, 1.150355339050293, -0.3547542095184326, 0.24675889313220978, -0.25303202867507935, 0.6037725210189819, 1.0986195802688599, -0.9196267127990723, -0.9423813819885254, 1.679168462753296, 1.0018244981765747, 0.4854460060596466, -0.5472642779350281, -0.6111990213394165, 0.1759767383337021, -1.1599138975143433, 0.7487642765045166, 0.7696691751480103, -0.8765849471092224, 0.7039768695831299, -0.18548819422721863, 0.9369907975196838, -1.5481809377670288, -0.1512017697095871, -0.2100934237241745, -1.419253945350647, -0.41555094718933105, 1.269731879234314, 1.0553160905838013, 0.7093477249145508, 2.330687999725342, -1.4964765310287476, -0.34602090716362, -0.2689455449581146, -0.5926432609558105, -1.8359588384628296, 1.569271206855774, 1.417257308959961, 0.5024587512016296, 0.9541721940040588, 0.7802327871322632, -0.16833660006523132, -1.2843348979949951, 0.1772257387638092, -0.1506900191307068, -0.22270382940769196, -1.7101900577545166, -0.5831116437911987, -0.8230295777320862, 0.4643067419528961, -0.7439783811569214, -0.8587275743484497, 0.24796317517757416, 0.7978287935256958, -0.6895995736122131, 0.3962307274341583, -1.6630663871765137, -0.2857015132904053, -0.28069183230400085, 0.5953629016876221, -1.0314921140670776, 0.38079139590263367, -0.7947635054588318, -0.0035220065619796515, -1.2985316514968872, -0.11121950298547745, 1.0948824882507324, -1.0238789319992065, 0.898669958114624, 1.9069857597351074, 1.3713653087615967, -1.2684246301651, 0.3176015019416809, -0.8159079551696777, -0.6554272770881653, 0.2637793719768524, 1.0340261459350586, 0.649355411529541, 1.3048458099365234, 0.2894676923751831, -2.010361909866333, -1.3369742631912231, 1.1196821928024292, 0.5690990090370178, 0.6597016453742981, -1.0160839557647705, -0.6917330622673035, 0.06214065104722977, 1.3951088190078735, -2.2858076095581055, 0.47243058681488037, 0.740077555179596, -0.18247312307357788, -0.5681853294372559, 1.233309268951416, -0.9196562767028809, 0.101236991584301, 0.6411806344985962, -0.42539069056510925, 1.3914071321487427, 1.54865300655365, 0.3124178647994995, 0.560121476650238, -1.4682788848876953, 0.7813104391098022, -1.1563045978546143, -0.1951206773519516, -0.6573917865753174, -2.3869619369506836, 0.39637500047683716, 0.4922325909137726, -0.1090352013707161, -1.0362529754638672, -0.22967472672462463, -1.3549021482467651, 2.1005635261535645, 0.44912803173065186, 2.4793853759765625, -0.5752666592597961, -1.4330936670303345, 0.005707933567464352, 0.09736598283052444, -1.583479642868042, -0.6063992977142334, -0.8874954581260681], [0.9648095965385437, -0.2949208617210388, 1.287294864654541, -0.791971743106842, 0.34913378953933716, -0.20917731523513794, -0.5044708847999573, 0.16700029373168945, -0.9490258097648621, 0.3087582588195801, 0.8026031255722046, 0.6036773920059204, 1.1904730796813965, -1.1836680173873901, 1.6909277439117432, -0.4697629511356354, 1.5072007179260254, 2.2194759845733643, 0.32584452629089355, 0.331037312746048, -0.7652121186256409, 1.073730707168579, -0.037393711507320404, -0.1495465785264969, 1.1237828731536865, -0.6727063059806824, -0.7538083791732788, -1.6924622058868408, 0.6084914207458496, 0.3165377974510193, -0.36503997445106506, 0.075379878282547, 1.002734661102295, -0.24007244408130646, 0.7427836656570435, -0.38940805196762085, 0.518052875995636, -2.5261361598968506, -0.24014140665531158, 0.22367709875106812, 0.9566092491149902, -1.089438796043396, -0.8222568035125732, -0.272165983915329, -0.23375184834003448, -0.5293434858322144, 0.7319176197052002, 0.5569950938224792, 1.0770456790924072, 0.27063360810279846, -1.019597053527832, 0.11429926753044128, 0.017176317051053047, -0.1126563772559166, 1.6396043300628662, 0.6446216106414795, 1.1426280736923218, 0.7973176836967468, -0.6093605160713196, -1.0360431671142578, -0.1191740334033966, 0.8897333145141602, -0.24724382162094116, 0.28405165672302246, -0.00972782913595438, -0.8765720725059509, -0.4309171140193939, 0.5022003650665283, 0.20391634106636047, 0.8619138598442078, 0.8798220157623291, 0.04766269028186798, -0.9116744995117188, -1.2792243957519531, -0.6222596168518066, 0.34283971786499023, -1.6029618978500366, 1.3264281749725342, -0.4756699204444885, -1.725083351135254, 0.174315944314003, -2.270921468734741, 0.33338963985443115, -0.5447578430175781, 0.23649761080741882, -1.1426129341125488, -0.5274553894996643, 0.15010179579257965, 0.12038663029670715, 1.5737135410308838, 1.2145789861679077, 0.24343359470367432, -0.4502533972263336, -0.7276486158370972, -2.015972375869751, 0.221767395734787, 1.250754714012146, 0.8012605905532837, 2.1285173892974854, 0.6170642971992493, -0.17607201635837555, 0.7867603898048401, 0.4072720408439636, -0.2623313367366791, -0.6844518780708313, -0.8334383368492126, -0.6849116086959839, 0.14211004972457886, 0.20015381276607513, 0.41604670882225037, 1.2961430549621582, 0.33170294761657715, -0.14009897410869598, -0.7255426645278931, 0.28589892387390137, 0.212804913520813, 1.2344708442687988, -0.22188879549503326, 2.2951440811157227, -0.3325732350349426, -0.13559593260288239, 0.730519711971283, -0.008544174022972584, -1.1136887073516846, -0.5976006388664246, -0.2982213795185089, -0.630843997001648, 0.5359531044960022], [0.10410308837890625, 0.41805511713027954, -1.2081586122512817, 0.27646028995513916, -0.05768362805247307, 0.6584112644195557, -0.1537162810564041, -0.5215761661529541, 0.21244022250175476, -0.34129634499549866, 0.42631202936172485, -0.031009508296847343, -0.6947131156921387, 0.8315953016281128, -1.077690839767456, 0.09538283199071884, -0.35508501529693604, 0.595214307308197, 0.5196414589881897, -0.20037993788719177, 0.8416836261749268, -0.3211188316345215, 1.1166023015975952, -0.595043957233429, -2.341667890548706, 0.033363547176122665, -1.0709202289581299, -1.653079628944397, -0.48091939091682434, -0.36889731884002686, 0.5192261934280396, 0.06277373433113098, 0.9323911666870117, 0.8524130582809448, 0.8010227084159851, -2.5216012001037598, 1.1140036582946777, 0.46953266859054565, 0.6799096465110779, -1.155761957168579, 0.7958563566207886, -1.656325340270996, -0.2395617663860321, 0.01693783327937126, -0.7645432949066162, 0.8185005784034729, -0.9539774656295776, 1.2195262908935547, -0.2542710602283478, -2.2557315826416016, 0.5111193060874939, 0.3167586624622345, -0.36863237619400024, 0.3552379608154297, 1.2428488731384277, -0.12673752009868622, -0.7079582810401917, 1.7520779371261597, 0.3023397922515869, -0.6745935678482056, -1.6234465837478638, -0.10968201607465744, 1.1823232173919678, -0.7494341731071472, 1.5231043100357056, 2.1283795833587646, 0.35996779799461365, 1.0329197645187378, 2.788414716720581, -0.5795786380767822, -1.7122080326080322, -2.345391035079956, 0.4810921847820282, -2.361109733581543, -0.23880895972251892, -0.981644332408905, 2.2614822387695312, -1.7115696668624878, -1.884422779083252, 0.5432775020599365, 0.7834085822105408, 0.6868998408317566, 1.8727608919143677, -1.0688501596450806, -0.661939263343811, -0.7052001357078552, -0.4482460916042328, -1.727548360824585, -0.47925612330436707, -0.7065717577934265, -0.027886228635907173, -0.7801045775413513, -0.7883492708206177, 2.1750869750976562, -0.9782225489616394, 0.33935850858688354, -1.6393482685089111, -0.11643866449594498, 1.2983063459396362, 1.522130012512207, -1.9361766576766968, -1.9755414724349976, -1.573994517326355, 0.6330022215843201, -1.277153730392456, 1.0165929794311523, 0.9461311101913452, 0.11739934235811234, -0.921757698059082, -0.05729811638593674, -1.72416090965271, -0.34512224793434143, 0.29519328474998474, -0.556522011756897, 0.14935694634914398, -0.5763130784034729, 0.7762840986251831, 0.05141830071806908, 0.42533621191978455, 0.856313169002533, -2.0955698490142822, -0.08718381822109222, -0.3078761100769043, 0.19452665746212006, -1.79063880443573, 0.46951717138290405, 0.3452930748462677, -0.956649899482727], [-1.0649725198745728, -1.0811667442321777, 1.5642937421798706, -0.18570442497730255, 0.2802399694919586, -0.5872596502304077, -0.07899268716573715, -0.6370261311531067, -0.4025396704673767, 1.015782356262207, -1.3476676940917969, 0.6200904846191406, -0.048138782382011414, -0.667415976524353, 0.18226255476474762, 1.3191378116607666, -0.8925489783287048, 0.7743829488754272, 0.1991388499736786, 0.4390513598918915, 2.25982928276062, 0.0729527622461319, -1.1830847263336182, -1.0851843357086182, 0.30583810806274414, -0.4943302571773529, -1.1101300716400146, -0.07760310173034668, 0.4049929082393646, 2.035799741744995, 1.4186912775039673, 1.7344717979431152, -0.49753978848457336, -0.5845958590507507, 0.04151991009712219, -0.25757917761802673, 1.87935471534729, 0.5671845078468323, 0.6960716247558594, -1.816041350364685, 0.9150491952896118, 1.1635193824768066, -0.9474005699157715, 0.7598690390586853, -0.7697198390960693, -0.8552563786506653, -0.38709813356399536, 0.6449278593063354, 0.052494682371616364, 0.8790873289108276, -0.5390637516975403, -0.6965205669403076, -1.9588662385940552, 0.8217646479606628, -0.18857726454734802, -0.18015676736831665, -1.0652518272399902, -2.129948616027832, 0.5115225911140442, -1.8781485557556152, -0.39599987864494324, 0.4544714391231537, -2.276698589324951, 0.31451621651649475, -1.3246186971664429, -1.4845006465911865, 0.25935861468315125, 1.4269676208496094, 0.9486919045448303, 0.2302495688199997, -1.6465539932250977, 0.6549002528190613, -0.5377961993217468, 0.4857359528541565, -0.10677142441272736, -0.17352566123008728, -0.6614877581596375, 0.21328303217887878, -0.18122662603855133, -1.013819694519043, -1.7110341787338257, -0.924430251121521, 1.4167522192001343, 0.304799348115921, -1.5349456071853638, 2.4647395610809326, -2.0523746013641357, -0.5719922780990601, 0.9940799474716187, -0.18875250220298767, -0.43475955724716187, 0.23087556660175323, 0.32064470648765564, 0.04668625071644783, -1.0548449754714966, -1.30502450466156, 0.13015125691890717, 0.6261842846870422, 0.9907397627830505, 0.7978368401527405, -1.955872654914856, -1.338505744934082, 1.4773274660110474, 1.8460220098495483, 1.094017505645752, -1.4135950803756714, 0.9980933666229248, -1.5189661979675293, -0.3823845088481903, -1.015572190284729, 0.6439841389656067, -0.12217757105827332, 1.1631346940994263, -0.8215912580490112, -0.8673005700111389, 0.920813262462616, -0.59124356508255, 0.35500937700271606, -1.442662239074707, -0.9424843192100525, 1.8039076328277588, 0.6034761071205139, -0.4129537045955658, -0.9704292416572571, 0.8496685028076172, -1.0442731380462646, 0.2608262002468109, 1.4625376462936401], [-0.11266493797302246, -0.9998359680175781, 1.1932274103164673, 0.5964768528938293, -0.012968736700713634, 0.18964321911334991, -1.031282901763916, -1.4139368534088135, -2.1579737663269043, -0.8913447856903076, 0.2882952094078064, 0.39849668741226196, -0.01959003508090973, -1.6830390691757202, 0.3829966187477112, 1.2434308528900146, 0.03902226313948631, 0.38216760754585266, 1.6202285289764404, -0.5879173278808594, -0.8229283690452576, 1.3122897148132324, -0.1654539406299591, -1.4648818969726562, 0.09495710581541061, 1.0541765689849854, -0.8649938702583313, -0.41207849979400635, 1.2672250270843506, 0.04881976917386055, 2.4099955558776855, 0.5179744362831116, -0.32646575570106506, -0.6653891205787659, -0.2585446238517761, 0.03691617771983147, -1.0280778408050537, 0.0462300069630146, 0.16226054728031158, 0.809756875038147, -1.9555180072784424, 1.1639022827148438, -0.4209774136543274, -1.4169297218322754, 0.21626362204551697, 0.681405246257782, -1.2160152196884155, -1.3844351768493652, -1.3880864381790161, -0.8390698432922363, -0.0653114914894104, 1.323264718055725, -0.21025502681732178, 1.0217341184616089, 1.8641995191574097, 0.8170140981674194, 0.5815935134887695, 1.8245655298233032, 0.7685457468032837, 0.313951700925827, -1.0220987796783447, 2.0809051990509033, 0.31932052969932556, 0.4135103225708008, -0.736309826374054, -0.8924290537834167, 0.4526168406009674, -0.8565366864204407, -0.07905468344688416, 1.0795834064483643, 0.3804086446762085, -0.22044752538204193, -0.11367377638816833, 2.1965255737304688, 0.622355580329895, 1.8610420227050781, 1.1695584058761597, 0.8516213297843933, -0.7040796279907227, 1.272274136543274, 0.5731298327445984, -1.8962604999542236, 0.6437993049621582, 1.1545968055725098, 2.9864418506622314, 0.7928082346916199, -0.2976624071598053, 1.049790859222412, 0.21640750765800476, 0.5245945453643799, 0.6609926223754883, 0.7411110997200012, -1.3381017446517944, -1.1662101745605469, -0.036579493433237076, 0.7104952931404114, -0.8444367051124573, -0.7686542272567749, -1.608546495437622, 1.5509397983551025, 0.0008953072829172015, -0.7478616833686829, -1.1116775274276733, -0.40366116166114807, -0.8612184524536133, 0.348426878452301, -0.9391244649887085, 0.3326989710330963, -2.6506783962249756, 0.041402265429496765, -2.4705843925476074, 1.076810598373413, 0.8109648823738098, -1.0534400939941406, -0.3534112572669983, -0.4148830771446228, -0.3385302722454071, -1.2554128170013428, -0.7061460614204407, -0.22433629631996155, 0.42840251326560974, 0.25598546862602234, 0.2045057713985443, -0.2894718647003174, -0.6626773476600647, -0.2668885886669159, 0.9368427395820618, -1.1005069017410278], [0.14756222069263458, -1.756488561630249, -1.998317003250122, -1.5183323621749878, 0.5859243869781494, 1.6134583950042725, 0.43016910552978516, 0.3407115340232849, -0.8089923858642578, -2.0417869091033936, 1.1442577838897705, -0.3572036921977997, -0.6574452519416809, -0.8213720917701721, 0.4663980305194855, -0.1453951746225357, -1.6439787149429321, 1.0158125162124634, 0.5166407227516174, -0.1265036016702652, -1.0899252891540527, -1.1511199474334717, 0.8186179995536804, -0.3307734727859497, -0.7120373845100403, -0.5576993823051453, -1.05972421169281, -0.00929571408778429, -0.549187421798706, 1.4532824754714966, 0.24464458227157593, -0.6309264898300171, 0.265250027179718, 2.780128240585327, 0.7097545862197876, -0.30598336458206177, 0.4238748550415039, 0.7568396925926208, 0.15804535150527954, -1.0974557399749756, -1.1014611721038818, 1.0536943674087524, 1.6835181713104248, -0.5084490776062012, 0.6874261498451233, -0.38517287373542786, -0.9207248687744141, -0.5222784876823425, 0.6804473400115967, -0.14657093584537506, 1.6797173023223877, 0.051642537117004395, 0.251486212015152, 1.1258838176727295, 0.8373645544052124, 0.4350598156452179, -1.5344021320343018, -0.7424403429031372, -0.5115469694137573, 1.2337158918380737, 1.9590039253234863, -0.269197016954422, -1.6601389646530151, 0.8790742754936218, 0.6385685205459595, 0.08385740220546722, -0.059855975210666656, 0.5803107619285583, 0.33514299988746643, 1.4372413158416748, -0.24910376965999603, 1.1383185386657715, 0.04971575364470482, 0.22599245607852936, -1.399324893951416, 1.3275266885757446, -0.2645282447338104, 2.1456058025360107, -0.04050830379128456, 0.8246050477027893, 0.5944820046424866, 0.7414475083351135, 0.14113812148571014, 1.219852328300476, 0.28645387291908264, -1.0633383989334106, 1.0491400957107544, -0.5618233680725098, 0.08243533968925476, 2.140413522720337, -1.8659133911132812, -0.6032038927078247, -0.439558207988739, -1.1292626857757568, -1.0316987037658691, 0.3508196473121643, 0.4653181731700897, 0.6281521916389465, -0.7895999550819397, -1.5276058912277222, -0.30645954608917236, -0.18614356219768524, 0.7699382901191711, -0.44809988141059875, -0.47059470415115356, -0.5603837370872498, -1.5963859558105469, -0.4420664310455322, 1.2615076303482056, 1.2843267917633057, 0.1926543116569519, 0.500058650970459, -0.38413992524147034, -0.3487975597381592, -0.44347864389419556, -0.2837411165237427, 0.6646482944488525, -0.48662734031677246, 1.1156796216964722, 1.9734854698181152, -0.2521001696586609, -0.8118206262588501, -0.32882654666900635, 0.6934378743171692, -1.287653923034668, 1.070525050163269, -0.06432486325502396, -0.20067769289016724], [-1.8266206979751587, 0.2736470699310303, 0.1974164843559265, 0.07289745658636093, 0.2199956774711609, -0.5784772634506226, -0.7653297781944275, -0.3786073625087738, 1.3794169425964355, 2.1065633296966553, -0.8351499438285828, -0.9351839423179626, -1.1139423847198486, 1.3339383602142334, 1.006820559501648, 0.4944870173931122, -0.5127921104431152, -0.3523416221141815, 0.7857489585876465, 0.3533872365951538, -0.1868741363286972, 0.08996418863534927, -1.6862682104110718, -0.1092061772942543, -0.365820050239563, -1.2299561500549316, 0.33194589614868164, 3.0679070949554443, 1.6862847805023193, 0.6507308483123779, -1.5062919855117798, 0.0008892926853150129, 1.252263069152832, -1.6585091352462769, 1.793784499168396, 1.5513356924057007, -0.5518672466278076, 0.20021355152130127, -0.7173019051551819, -0.9067236185073853, -0.8182609677314758, 1.2808743715286255, -1.6999279260635376, -0.05118729919195175, -0.10257872939109802, 0.14407862722873688, -0.940605878829956, 1.1451160907745361, -1.4863567352294922, -0.9599628448486328, -0.02324517071247101, 0.195995032787323, 0.7537611126899719, 0.14879892766475677, 0.5586304068565369, 0.259619802236557, 0.14371521770954132, -0.09998616576194763, -0.8468367457389832, -1.3809494972229004, -1.9962782859802246, 0.3079468607902527, 0.00014981644926592708, 0.43732696771621704, 0.09317848831415176, -1.3287187814712524, -0.13884702324867249, 0.44899943470954895, 0.1952906847000122, -0.8259063363075256, 1.1041603088378906, 1.552138328552246, -0.1366511583328247, -1.958500623703003, -0.8653796911239624, 1.0135526657104492, -1.8240160942077637, -0.6251847147941589, -1.591975450515747, -0.20957808196544647, -1.4616276025772095, -0.37047508358955383, 0.8899039626121521, -1.3073428869247437, 0.6325360536575317, 0.07947264611721039, -1.4170645475387573, 0.6012691259384155, 0.6026713848114014, -0.16142958402633667, 0.5207289457321167, -1.2009937763214111, -1.581918478012085, 0.38961857557296753, 1.67043936252594, 1.2842472791671753, 0.7217782139778137, 0.7175482511520386, -0.6713602542877197, 0.31736835837364197, -0.7702494859695435, -1.0634950399398804, 1.4951941967010498, -1.131585717201233, 1.5878281593322754, 0.3362096846103668, 0.6345281004905701, -1.2743406295776367, 0.23083829879760742, -0.8096376657485962, -0.2905203402042389, -0.8753916621208191, 0.5826163291931152, 0.3329046368598938, -0.398441880941391, -0.263047456741333, -1.715826392173767, -0.3852766752243042, 0.6293668746948242, 1.4052776098251343, 0.22062039375305176, 1.1300731897354126, -0.07500334829092026, -1.8794963359832764, -0.02011132799088955, 0.08359727263450623, 1.1767112016677856, -1.28414785861969], [2.2584457397460938, 0.719716489315033, 1.545499324798584, 1.4400880336761475, 1.7616405487060547, -1.2931430339813232, 0.3044625222682953, 0.3434430658817291, 1.1326707601547241, -1.5744011402130127, 0.6126816272735596, -0.751674234867096, -0.5723467469215393, 0.12641222774982452, -0.46349257230758667, 0.6712644100189209, -0.5040537118911743, 0.4979339838027954, -0.7214226722717285, -0.9390572905540466, 1.0711185932159424, -0.19771263003349304, -1.058450698852539, -0.18281692266464233, 0.19811351597309113, -0.2514105439186096, -0.9492900967597961, -1.3713829517364502, 1.6246004104614258, 0.48190057277679443, 0.20630861818790436, 1.7213184833526611, -1.3517965078353882, -1.6237653493881226, -1.3583440780639648, 1.2153213024139404, -1.6891717910766602, -1.8717756271362305, -0.9307138919830322, -1.0907584428787231, 0.44724100828170776, 0.22637884318828583, 0.8903431296348572, 0.26713326573371887, 0.09127204865217209, 0.30070239305496216, -1.8768084049224854, -0.2954253852367401, 1.2389837503433228, -0.28554224967956543, -1.243493914604187, 0.14831092953681946, -0.24031750857830048, 0.16652843356132507, 0.8738677501678467, -0.5355395674705505, 0.9699069261550903, 0.5723503828048706, -1.0268454551696777, 0.6222113966941833, -0.919651985168457, -0.9028031229972839, 0.0846688374876976, 0.5498903393745422, 0.5064373016357422, 0.33786725997924805, -0.8314858078956604, -1.1785908937454224, 0.2542344629764557, -1.3644739389419556, -0.3991023600101471, 0.08103025704622269, 1.8610965013504028, 0.022994382306933403, 0.31265154480934143, 0.8585863709449768, 0.2501020133495331, 0.9139604568481445, 0.23337148129940033, 0.8943371772766113, 0.3536342680454254, -0.4822513163089752, -1.31620454788208, -0.09113487601280212, -1.025866150856018, -1.1885993480682373, -0.34851109981536865, -1.812669277191162, -0.3094046413898468, 0.5648342370986938, -0.5474462509155273, -0.3515218496322632, -0.46617963910102844, -0.06945651769638062, 2.283256769180298, 0.7937182784080505, 0.6306225657463074, -1.5507378578186035, 0.45123741030693054, 1.1045857667922974, -1.205280065536499, -1.7465696334838867, -0.10737957805395126, 0.22623282670974731, -1.2033015489578247, -0.49172258377075195, -1.5212973356246948, -0.6000126600265503, -1.6631439924240112, -1.3476314544677734, 1.2625831365585327, 1.9894121885299683, 0.7114508152008057, 1.6562548875808716, -1.568019986152649, 0.5999391674995422, 0.7315685153007507, 0.7910407781600952, -0.40738028287887573, 0.19438882172107697, -0.5667858719825745, 1.2056457996368408, -1.0662240982055664, 0.04220876097679138, -0.5875095129013062, -0.16535310447216034, 1.0585232973098755, -1.5075957775115967], [-0.8658073544502258, -0.14907285571098328, -0.43631190061569214, 0.16276532411575317, 1.0294888019561768, -0.45318111777305603, -0.5197022557258606, 1.408582091331482, 0.22001108527183533, -2.024521827697754, 1.101718544960022, 0.6318673491477966, 0.7780770659446716, 0.02488042786717415, 0.30743110179901123, 0.21579059958457947, -0.9851107597351074, 1.4099725484848022, 1.2207547426223755, -1.0391370058059692, -0.3532388210296631, 0.6198177933692932, -0.8752499222755432, 0.09258566051721573, 1.2687073945999146, -1.2288058996200562, 1.1802210807800293, -0.09466078877449036, 0.8676087856292725, -0.8265783190727234, -1.7259225845336914, -0.1837199330329895, 0.10590910911560059, 0.8057875037193298, -1.2691997289657593, -0.590634822845459, 0.8543539047241211, 0.7464165687561035, 1.0477111339569092, -0.1158503070473671, 2.1316587924957275, 0.9357845783233643, -0.0037085069343447685, -0.21345943212509155, -0.5859249830245972, -1.6570252180099487, -0.7311909794807434, -0.8436319828033447, 0.5681130290031433, -0.41867873072624207, 1.0071756839752197, 0.6158161759376526, 0.6905049681663513, 0.47829508781433105, -0.8060135245323181, 1.3808178901672363, -1.2687361240386963, 1.1741876602172852, 1.3669949769973755, 0.910146176815033, 0.7750175595283508, 0.5354381799697876, -0.40778428316116333, -0.20986537635326385, 0.028381304815411568, 0.19543810188770294, 0.7282541990280151, 0.619904100894928, -0.5993713736534119, -1.4733952283859253, -0.7137997150421143, -1.544934630393982, 0.867165744304657, -0.1949915885925293, 0.06377548724412918, -1.2628895044326782, 0.6368908882141113, 1.6053941249847412, 1.564163327217102, 0.6231932640075684, 0.794434666633606, -0.6726589202880859, -0.05583646520972252, -0.163417249917984, 0.9433929920196533, 0.33177414536476135, -1.3944716453552246, 0.709028959274292, 0.08126184344291687, -1.2586122751235962, -1.0074331760406494, -0.004425701219588518, 1.1995865106582642, 0.5982969999313354, 0.25007542967796326, -0.41699013113975525, -0.06837523728609085, 1.0050742626190186, -0.8611849546432495, -1.425872564315796, 0.29291337728500366, 1.1866724491119385, 1.6624170541763306, 0.38555651903152466, -0.553127110004425, 0.2158263623714447, -1.062371015548706, -1.1209381818771362, -0.07332668453454971, 0.46772435307502747, 0.3621155321598053, 0.3661459982395172, 0.1877431571483612, -0.45608341693878174, -1.3457515239715576, -0.17081904411315918, -1.6308492422103882, -1.3118460178375244, -0.7191922068595886, -0.3210601806640625, 0.502600908279419, 0.7560752034187317, -0.2977782189846039, 0.2718418538570404, 0.1824020892381668, 1.5930702686309814, 0.05200079083442688, -1.1145356893539429], [-0.9104647636413574, -0.5297331809997559, 0.7767089009284973, -0.34060025215148926, -0.1739197075366974, -1.3998326063156128, -1.6859452724456787, -2.1531014442443848, -2.1298351287841797, 0.29650959372520447, -0.3658612072467804, 0.9947973489761353, -0.5479015707969666, 0.0709971934556961, -0.44172272086143494, -0.8561437129974365, 0.8103192448616028, 0.6553829908370972, -0.29796943068504333, -0.36919811367988586, 0.9661928415298462, 0.5078933238983154, 0.0005855046911165118, 0.41718170046806335, -0.5845571160316467, 1.1954036951065063, 0.7909828424453735, -0.48398998379707336, -0.5728192925453186, 0.022903723642230034, -1.1551673412322998, 0.8619862794876099, -0.08194328844547272, 1.0440236330032349, 0.364721417427063, -1.358120083808899, 0.8022478818893433, 0.44430872797966003, -0.4522494375705719, -1.1170798540115356, -1.1071990728378296, 0.006908607669174671, 0.10034039616584778, 1.0471889972686768, 1.4334298372268677, -1.452658772468567, -2.369534730911255, -0.23515786230564117, 0.6554028391838074, -0.9045606255531311, -0.44115281105041504, -0.23144720494747162, -0.1660090833902359, -0.49486005306243896, 0.2718406915664673, 0.7324798703193665, -0.5744054913520813, 0.07629416137933731, 1.1442487239837646, 2.129568099975586, -1.3258899450302124, 1.0729576349258423, 0.43842899799346924, -0.5270740985870361, -0.8712998628616333, 0.14068563282489777, -1.3158437013626099, 1.4755345582962036, 0.6845932602882385, 1.402442455291748, 1.515106439590454, -0.4578952491283417, 1.2143034934997559, -0.48313960433006287, -1.341995120048523, 0.1157468631863594, 0.4500335156917572, 0.279207706451416, -0.7163474559783936, -2.1681933403015137, 1.2425010204315186, 0.8252659440040588, 0.4232892096042633, 1.0285279750823975, 0.9786748290061951, -2.917762041091919, -0.37847182154655457, -0.9911680221557617, -0.5787078738212585, -0.5752539038658142, 0.8232820630073547, 1.3518881797790527, 0.3688737154006958, 1.3870387077331543, -0.03822529688477516, -0.9857311844825745, -0.545979917049408, -1.860626220703125, -0.700087308883667, -0.9193523526191711, -0.7693521976470947, 0.43070027232170105, -0.31587621569633484, -1.1569288969039917, -1.3432872295379639, 0.11343150585889816, 0.9117504358291626, -1.1108119487762451, 0.31941235065460205, 1.251198649406433, 2.3038241863250732, -0.4502599537372589, 0.9997724294662476, -0.6438467502593994, 0.8417339324951172, 0.47112083435058594, -0.7408431172370911, 1.0158674716949463, -0.5342807769775391, 0.6624866724014282, 0.36125674843788147, 0.07490154355764389, 0.8478748202323914, -1.77129065990448, 0.24427253007888794, 1.5793639421463013, -1.321338176727295, 1.2520142793655396], [0.9811595678329468, -1.4196833372116089, -0.6837161183357239, 0.46958979964256287, 0.044723883271217346, 0.062406644225120544, 1.1580561399459839, 0.6443729400634766, -0.3885575830936432, -0.7183971405029297, -0.43438366055488586, 0.7125962376594543, 1.663394808769226, -0.8120887279510498, 0.48418110609054565, 0.44212082028388977, -0.027944523841142654, 0.4871971011161804, -0.016405774280428886, 0.7025940418243408, 1.0798622369766235, -0.25015273690223694, -1.0711593627929688, -1.0797209739685059, -0.8466324210166931, -1.218333125114441, -1.2194839715957642, -0.007891084998846054, -0.06312929093837738, -1.5888413190841675, -1.2392557859420776, 1.0005083084106445, 0.49214908480644226, 0.06672839820384979, 0.8640691041946411, -1.1503795385360718, -0.30739012360572815, 0.7510876655578613, -0.4927622079849243, -0.2320139855146408, -1.1058037281036377, -1.0373154878616333, 0.1928325742483139, 0.33318030834198, -1.0070332288742065, -0.48812395334243774, -1.8974640369415283, -0.9452022910118103, 0.17610754072666168, 0.30095139145851135, -1.0559916496276855, 0.37692978978157043, 1.6889601945877075, 0.3553306460380554, 0.7970554232597351, 0.2821429967880249, -2.368090867996216, -0.2629470229148865, -0.6623579859733582, -0.5418910980224609, -1.1374092102050781, 0.9459249377250671, -0.37441202998161316, -0.4824778735637665, 1.804599642753601, -0.35354122519493103, -0.4680604934692383, -0.6391445994377136, -0.271132230758667, -3.030207872390747, 0.02121635712683201, 0.5473207831382751, 0.15225467085838318, 1.472144365310669, 0.35478994250297546, -0.5910152196884155, 0.4664137363433838, -0.9987773895263672, 0.7435832619667053, 1.8155031204223633, -0.0748421773314476, -0.12904593348503113, -0.011966477148234844, 1.1934534311294556, 0.6014008522033691, 0.5702534914016724, 0.3342611789703369, 1.0812885761260986, -0.6028545498847961, 0.41166892647743225, 0.19711479544639587, -0.7572340369224548, 0.8509772419929504, -0.939834475517273, 0.597562849521637, 0.08730323612689972, 1.9627240896224976, -0.2547740638256073, 0.10142695903778076, -0.39880961179733276, -1.1482431888580322, -0.14908446371555328, 0.6586464643478394, 0.7799766659736633, -2.612487554550171, 0.5127801895141602, 0.4238651990890503, -0.7212640643119812, -0.9599864482879639, -0.3601917624473572, 0.7787762880325317, -0.060107216238975525, -0.989209771156311, 0.3416585624217987, -0.5660362839698792, 0.794161856174469, -0.90653395652771, 0.9134836196899414, 1.0714350938796997, 0.26946863532066345, 1.6046206951141357, 0.5918529033660889, -0.38803714513778687, 0.39793193340301514, -0.8600447177886963, 0.7611907720565796, -2.8103299140930176, -0.1351856291294098], [-0.1491370052099228, -1.4619704484939575, 1.8823212385177612, 2.249241352081299, 0.35704654455184937, 0.44543030858039856, -0.624301552772522, -1.7510313987731934, 0.9111815690994263, -0.7833273410797119, 0.3691873252391815, -0.4031387269496918, 1.2888814210891724, 1.1618752479553223, 0.12417841702699661, -0.2893272936344147, -0.15679287910461426, 0.5353083610534668, 0.8020332455635071, -2.4236738681793213, 0.7241044044494629, -0.33752578496932983, 0.8078498840332031, -1.356379508972168, -0.9197118878364563, -1.303173303604126, 1.579001784324646, -0.08220133185386658, -0.6128100156784058, 0.27134981751441956, 1.5617729425430298, 1.1707876920700073, -1.0934444665908813, 0.9559402465820312, 0.3857380747795105, 0.5673568844795227, -0.549008309841156, -0.6086229085922241, 1.808953046798706, -0.24009908735752106, 0.30441567301750183, 0.30260753631591797, 0.49649617075920105, -0.649394154548645, 0.7308369278907776, -0.09548389911651611, 0.640999436378479, -0.6705748438835144, 0.16815373301506042, -0.20981620252132416, -0.5429834723472595, 1.4508317708969116, 0.47712093591690063, 0.1499660611152649, 0.07325150817632675, -0.31164050102233887, -0.0880456417798996, 1.8128811120986938, 0.4910730719566345, 1.014730453491211, -0.0487215593457222, -0.048955850303173065, -2.0136313438415527, -1.1425509452819824, 0.7933687567710876, -0.22523649036884308, -0.4384666085243225, 0.4625057280063629, -1.3913735151290894, -1.0618906021118164, 1.645633339881897, 0.10051698982715607, -0.8240103721618652, 0.7249805927276611, -1.2477316856384277, -0.5762883424758911, -1.905778169631958, -1.149836540222168, 0.30712127685546875, 0.0774398222565651, -0.449325829744339, -0.5606898665428162, -0.020443519577383995, 0.7154061198234558, -0.6378291249275208, 1.1316452026367188, -1.2289433479309082, -0.7246579527854919, 0.29436251521110535, -0.7955834865570068, -0.2828541100025177, 1.1685030460357666, 0.9012662172317505, -0.042802032083272934, 0.4398089051246643, 1.802321195602417, 0.16653376817703247, 0.3092016875743866, -0.029546303674578667, -1.1660282611846924, -1.0024200677871704, 0.01498194970190525, -0.05033785104751587, 1.288622498512268, 0.7936789989471436, 1.6705235242843628, 0.11338990181684494, 0.29714062809944153, -0.5986618995666504, -1.2596709728240967, 0.9044061899185181, -0.7769901156425476, 2.062798500061035, -0.2069290578365326, 1.0808221101760864, -0.8567869663238525, 0.8263351321220398, -0.12076963484287262, -1.5471190214157104, -0.03036302514374256, -0.14701038599014282, -0.8011499047279358, -1.5100938081741333, -0.003508168738335371, -1.4193847179412842, -0.06649920344352722, 0.5462164878845215, -0.25979065895080566], [-0.724210262298584, -0.6249135732650757, -1.6962859630584717, -1.511947751045227, 0.41366320848464966, 0.807019054889679, -1.5825577974319458, -2.3853933811187744, 1.4075005054473877, 0.3788139224052429, -0.0050802831538021564, 0.304584801197052, 1.2031222581863403, 0.15064960718154907, 1.0599133968353271, -0.2649408280849457, 0.924005389213562, -0.7822390794754028, 0.248155876994133, -0.1825110763311386, 1.3516377210617065, -0.5806597471237183, 1.1039788722991943, -1.4335589408874512, 0.1996026635169983, -0.11556387692689896, 0.7082404494285583, -0.3049173951148987, 0.6646769642829895, 1.7287336587905884, -1.0164072513580322, -0.8800082206726074, 1.294896125793457, 1.2052960395812988, 1.2311663627624512, 0.012019955553114414, 0.8399214148521423, 0.3000200688838959, -1.6797478199005127, 0.10458926856517792, 0.9180344343185425, -0.837556004524231, -0.5856409668922424, -2.8114607334136963, 0.20135818421840668, -0.010018838569521904, 0.34196871519088745, -0.4002983570098877, 0.25664183497428894, -0.28305354714393616, -1.5448212623596191, 1.179274320602417, -0.08057665824890137, 1.3541682958602905, -1.2571583986282349, -1.9406989812850952, -1.446834683418274, 2.909457206726074, -0.8221254944801331, 1.761320948600769, -1.8535850048065186, 0.2992779314517975, -1.4114320278167725, -0.19246940314769745, 0.1837523728609085, 0.010196120478212833, -0.08926122635602951, 1.1929570436477661, -1.3665306568145752, 0.3301510810852051, -0.24452686309814453, -0.42529186606407166, 0.6241697072982788, -1.3649142980575562, 0.46339255571365356, 0.05446617305278778, -0.9790117740631104, -0.5114567279815674, 0.6676783561706543, 0.30410975217819214, 0.592513918876648, -0.3895128667354584, 0.4750294089317322, -2.0375192165374756, 0.004464444704353809, 2.8509521484375, 1.7740983963012695, 1.8109397888183594, -1.2647498846054077, -0.7116406559944153, 0.49971431493759155, -0.019686121493577957, 1.5649935007095337, 1.655333161354065, -0.8313153386116028, 0.28453636169433594, -0.2925979793071747, -0.39784446358680725, 0.06810946017503738, -2.1613101959228516, 0.7067722082138062, 1.9119611978530884, 0.824833869934082, -0.1990535706281662, 1.5069431066513062, 0.7909429669380188, -2.814054012298584, 0.5027386546134949, -0.415810227394104, 0.8288232684135437, 0.91844642162323, 0.43168413639068604, 1.5933079719543457, 0.3297702372074127, -0.3548380732536316, 1.7602232694625854, -0.19952064752578735, 1.7311843633651733, -1.1820961236953735, 0.356334388256073, -1.1774322986602783, -0.2924216687679291, -0.6459290385246277, 2.637514114379883, -0.591498076915741, 0.7260568141937256, 0.07603573799133301, -0.15025879442691803]], [[-0.1491931527853012, -2.723600387573242, 0.14034341275691986, 0.18114934861660004, 1.7606216669082642, -0.5308008193969727, 0.8479586243629456, -1.0139573812484741, 1.5881439447402954, -1.2162034511566162, 2.882096767425537, -0.18028731644153595, 0.29493221640586853, -3.0598769187927246, 0.03289276733994484, 0.4587326943874359, 0.6902979016304016, 1.1375126838684082, -0.8335013389587402, 0.3465495705604553, 0.877558171749115, -1.8446505069732666, -1.039265751838684, 0.1708545833826065, 0.5358225703239441, -0.4791911840438843, 0.53892582654953, 0.7486238479614258, 0.5951870679855347, 0.944441556930542, 1.132901668548584, 2.3142640590667725, -1.1915528774261475, -1.0626847743988037, 0.33607128262519836, -1.3994371891021729, -1.146924614906311, 0.8048018217086792, 0.13200165331363678, 2.0092084407806396, -0.21087487041950226, -0.4611562192440033, -0.9634709358215332, -0.6146712899208069, 0.017683131620287895, 1.7226201295852661, 0.25763943791389465, 1.0960122346878052, -1.6601004600524902, -0.048776380717754364, 0.10418334603309631, -0.36020031571388245, -0.6416012048721313, -0.9507429599761963, 1.7964727878570557, -1.5175108909606934, -0.37358230352401733, -0.4827733337879181, -1.6692659854888916, -0.5057381391525269, -0.3517686426639557, -0.6814179420471191, -1.2528553009033203, 0.26977139711380005, -0.08602141588926315, 1.3966869115829468, -0.4543505907058716, 0.1424078494310379, -0.44803816080093384, -1.714307188987732, 2.4356439113616943, 0.5660772919654846, 1.3915112018585205, -0.4553997814655304, 0.28015950322151184, -1.0505927801132202, -2.210597276687622, -0.5901999473571777, 0.3253386914730072, -1.0425047874450684, 0.8195679783821106, 0.8948797583580017, 0.17919358611106873, -0.49569177627563477, 0.9129974246025085, 0.2913285195827484, 1.6700584888458252, 0.15483029186725616, -0.4093132019042969, -1.1008104085922241, 0.7255944013595581, -0.18117652833461761, -1.305570125579834, 1.9027422666549683, 0.06335370242595673, 1.1781631708145142, 0.054591983556747437, -0.17086534202098846, 0.31136971712112427, 0.404520183801651, 0.2331288903951645, -0.6206125020980835, 0.035111092031002045, 0.11127255111932755, 0.6458486318588257, -0.4972419738769531, 1.1666364669799805, -1.33799409866333, 0.5181519985198975, 0.15184007585048676, 0.25314104557037354, -0.567825198173523, -0.3817930817604065, -0.23486855626106262, 0.3818621337413788, -0.5793585181236267, -0.5257527828216553, -0.18238328397274017, 0.20067480206489563, -0.14919322729110718, -0.6140449047088623, 0.9209868311882019, -0.3388573229312897, 1.1817249059677124, -0.4926671087741852, 1.0341531038284302, 0.5691871047019958, -0.5032326579093933], [1.0654882192611694, -0.44520384073257446, 0.0011861595558002591, 1.1769285202026367, 2.785506248474121, -0.1229119524359703, 0.0065079801715910435, 0.07061892002820969, -0.2617415189743042, -0.23107650876045227, -1.3303688764572144, -0.2991085350513458, -0.9250990748405457, -0.15777063369750977, -1.120251178741455, 1.2409451007843018, -0.027401011437177658, -0.871234655380249, 0.89914470911026, 1.6458169221878052, 0.6915668845176697, 0.7577195763587952, 1.7911925315856934, -0.3634980320930481, 0.5290787816047668, 0.620533287525177, -1.1686958074569702, -1.0805482864379883, -0.49485909938812256, 0.6755532622337341, 0.44843780994415283, 1.6067866086959839, -2.010824680328369, -0.8456065058708191, -1.2184921503067017, -0.20579245686531067, -0.10530033707618713, -0.3010634183883667, -0.8434767127037048, 1.121380090713501, -0.17773054540157318, 1.065850019454956, -0.6105843782424927, -0.29652756452560425, -0.9940998554229736, 0.21327504515647888, -0.023786524310708046, 0.5983050465583801, -1.3486077785491943, -0.552651047706604, 0.9062466025352478, -0.5430557131767273, -0.3292710781097412, -0.5269417762756348, 1.5648040771484375, -1.2963292598724365, -0.21907442808151245, 0.6438120007514954, -0.3711644113063812, -0.56185382604599, -1.2610228061676025, -0.9439774751663208, 0.9657939076423645, 0.9168999195098877, 1.5755435228347778, 0.9695945382118225, -0.23317895829677582, 0.12960000336170197, -0.8823639750480652, -1.078245997428894, 0.4230751693248749, -1.070491075515747, -0.031249143183231354, -1.7172526121139526, -0.6143820285797119, 0.8488307595252991, 0.09538538008928299, 0.19266961514949799, 1.3311537504196167, -0.5830046534538269, -0.3545113503932953, -0.2871295213699341, 1.475373387336731, -0.7606687545776367, 0.7584764957427979, 0.838858962059021, -1.120898962020874, 1.9431499242782593, 0.2268611639738083, -0.68062424659729, -0.2878553867340088, 0.08027898520231247, -1.8492748737335205, -0.3920350670814514, -0.17018792033195496, -1.4266936779022217, 0.44649943709373474, -0.28239089250564575, 0.4167342782020569, 0.9248862266540527, -0.1938474178314209, 0.4589042663574219, 0.5191860198974609, 0.08029969781637192, -1.5857274532318115, -1.5514919757843018, -1.3086142539978027, -0.6533926725387573, -1.6427401304244995, 0.5511133074760437, 0.7214766144752502, 0.6691862344741821, 1.1509320735931396, -1.6511949300765991, 0.9302816390991211, 0.9132335782051086, -1.0175501108169556, 0.17524904012680054, 0.6299987435340881, 0.3149893283843994, 1.1962119340896606, 0.36948564648628235, 0.4102129638195038, -1.3729591369628906, 1.8870669603347778, 0.6650540232658386, 0.805519700050354, 0.16434377431869507], [0.6244564056396484, 0.560002326965332, -2.2022039890289307, -0.24513384699821472, -1.4731383323669434, -0.6995742321014404, -0.224625825881958, 1.119187831878662, 0.7933093905448914, -0.3379330635070801, 0.7994054555892944, 1.9750354290008545, 0.8283147811889648, 0.7641223669052124, 0.5292574167251587, 2.5677645206451416, -0.022789902985095978, -1.066304326057434, 0.21571071445941925, -0.46862056851387024, -0.725099503993988, -0.7919829487800598, 0.6423741579055786, 1.460508942604065, -1.409122109413147, 1.132407307624817, 1.6261770725250244, -0.005765100475400686, 0.12178844213485718, -0.603282630443573, -0.7550631761550903, 1.0657379627227783, -0.032202210277318954, 0.9620018005371094, -1.5431263446807861, -2.0706586837768555, -0.035156041383743286, -0.06494613736867905, -0.0018692993326112628, -0.2587210536003113, 0.14294423162937164, -0.788449764251709, -0.8195188641548157, -1.0870765447616577, 0.3094947934150696, -0.5828223824501038, -1.1986521482467651, -1.345719814300537, -0.15968577563762665, 0.4294653534889221, 0.2411523312330246, -2.307791233062744, 0.7661487460136414, -0.6677924394607544, 2.2172884941101074, 0.18402788043022156, 0.6182763576507568, 1.0606298446655273, -0.030613694339990616, -1.6721147298812866, 0.47470560669898987, -0.02943974733352661, -0.9642027616500854, -0.3352106213569641, 0.8657103180885315, -2.6445257663726807, -0.4160696268081665, 0.31366321444511414, -0.315140962600708, 2.17425537109375, 1.5642625093460083, 2.617558240890503, 1.2729685306549072, -0.46600615978240967, -1.8876465559005737, -0.8891564607620239, -0.5391653776168823, -2.6701619625091553, 0.8359047770500183, 0.9598001837730408, 1.1401937007904053, 0.12995202839374542, 0.9509565830230713, 1.2627822160720825, -0.300772100687027, 2.069411039352417, 1.4049549102783203, 1.574939250946045, 0.20559793710708618, 1.2792768478393555, -0.06699288636445999, 0.3112342059612274, -0.411065012216568, 0.7289571762084961, -2.560270309448242, -1.2262202501296997, 0.42907804250717163, -1.302071452140808, 0.8218460083007812, -1.4063720703125, 1.1623364686965942, 0.48093441128730774, 0.48425769805908203, -0.5118521451950073, -0.8339831233024597, 0.7264813780784607, 0.35557234287261963, -0.5450859069824219, -0.8375826478004456, -0.3305600881576538, 1.3184353113174438, -0.05687379837036133, 2.165654420852661, 1.183701992034912, -1.3302381038665771, -0.7837039828300476, -0.591200053691864, 0.5204764604568481, -0.6137369871139526, 0.7476462721824646, 0.6998095512390137, 1.1242262125015259, 1.3159741163253784, 0.6259714961051941, 0.26228243112564087, 0.04270249977707863, -0.3323564827442169, -0.04568769410252571], [-0.1255260407924652, -0.2483312338590622, 1.0515190362930298, 0.30665701627731323, -0.7504348158836365, 0.15989626944065094, 0.29436591267585754, -0.040989942848682404, 1.1484614610671997, -0.25159963965415955, 0.11644653230905533, -0.42617037892341614, -0.6128898859024048, 0.18631891906261444, -0.0033173016272485256, -1.4641889333724976, 0.9662259221076965, -0.20621876418590546, -1.2484374046325684, -0.2320847362279892, -0.14241090416908264, -0.6172454953193665, -1.5511690378189087, -0.6823927164077759, 0.26506859064102173, -1.2195292711257935, 1.480137825012207, 0.6927444934844971, 1.0971006155014038, -0.7347821593284607, -0.5812551379203796, 0.2287779599428177, 0.4589880108833313, -0.5480796694755554, 0.42291852831840515, 1.6604241132736206, -0.817826509475708, -0.14077430963516235, -0.003850211389362812, -1.3979542255401611, -0.41608908772468567, 0.08570689707994461, 2.011105537414551, 1.4702198505401611, 0.6467748880386353, 0.06713468581438065, 0.19094271957874298, -0.13055314123630524, -0.21996815502643585, 0.2610361576080322, -1.6768454313278198, 0.3234778046607971, 0.8753862380981445, -0.3622390925884247, -0.25807496905326843, -2.3690905570983887, 0.3951031565666199, 1.3790465593338013, 1.828722357749939, -0.23590172827243805, 0.5777588486671448, 0.7060319185256958, -0.010373040102422237, 1.971893072128296, 1.0610677003860474, 1.0367388725280762, -0.24303865432739258, -0.04143710806965828, 1.030773401260376, 0.043246764689683914, -2.233856201171875, -0.07244550436735153, -0.07740101218223572, 0.6264290809631348, -1.209573745727539, 0.14496634900569916, -1.4591885805130005, -1.8418083190917969, -2.205538749694824, 1.4238362312316895, 0.8544243574142456, 2.1741814613342285, -0.25424420833587646, 1.0984214544296265, 1.4515653848648071, 1.591270923614502, 1.340582251548767, -0.6076294779777527, 2.4030604362487793, -0.02001769281923771, -0.8169773817062378, 0.5152010321617126, 0.7521247863769531, -0.715155303478241, 0.6450141668319702, -0.6875077486038208, 1.157793641090393, -0.24458415806293488, -1.1169780492782593, 0.5237895250320435, 0.5561850070953369, -1.6868548393249512, -0.9428256154060364, -1.0531623363494873, -0.9743099212646484, 0.3600311577320099, -2.106962203979492, -0.08816505968570709, -0.4111151695251465, -0.7132019996643066, -0.23697875440120697, 0.8002992272377014, -0.6706944704055786, 1.1334106922149658, -2.039602041244507, 2.0688021183013916, 0.776216447353363, -0.336617112159729, -2.831420660018921, 0.8142120838165283, 2.159376382827759, 0.2875169515609741, -1.949579119682312, -1.0098381042480469, -0.866178035736084, 0.2298029661178589, 1.4219754934310913, 0.7212263941764832], [-1.4425679445266724, 0.7757944464683533, -0.417195588350296, -0.07100087404251099, -0.854896605014801, 0.2976458966732025, -0.45304715633392334, -0.22644631564617157, 1.101658582687378, 0.77008056640625, 1.0127297639846802, 0.3571564555168152, 0.7934132814407349, -0.43289652466773987, 0.8832012414932251, 0.288896769285202, -0.5904221534729004, 0.8416600823402405, -1.462851881980896, -1.7110317945480347, -0.07380323112010956, -1.1332018375396729, 0.8919666409492493, -0.05936955660581589, -1.382189393043518, 0.2806369364261627, 1.846583366394043, -1.1443297863006592, -0.2729923725128174, 0.9644069075584412, -1.878820538520813, 0.2598552405834198, -1.4276803731918335, 0.6769492626190186, -0.8564165830612183, 0.10462016612291336, -0.01858106441795826, 0.1317707598209381, 1.4878203868865967, -1.5130864381790161, 1.0785534381866455, 0.7492492198944092, -1.317929744720459, 1.3001434803009033, -0.20113423466682434, 0.7837710380554199, -0.1278039664030075, -1.7692621946334839, 1.622787356376648, 0.8016353845596313, 2.2330572605133057, -1.3006595373153687, -0.35462871193885803, 0.008746262639760971, -1.439471960067749, 1.1452094316482544, 0.8829689621925354, -0.001926289638504386, -0.007285784929990768, 0.21156306564807892, 0.6961507797241211, 2.083916664123535, -0.35919189453125, -0.3701980710029602, -1.479514241218567, 0.9718222618103027, 1.2460229396820068, 1.2758958339691162, -1.7062417268753052, -1.3061435222625732, -0.2184477001428604, -0.628551721572876, 0.03134969621896744, -0.09140412509441376, -2.3734984397888184, 0.852217972278595, 0.8915342092514038, 0.7650887370109558, -1.5763105154037476, -2.3779566287994385, -0.5214527249336243, -1.1581324338912964, 0.5193041563034058, 1.5732495784759521, -0.9714948534965515, -1.0092220306396484, 1.2016617059707642, 0.505903959274292, 0.3815787434577942, -0.9095214605331421, -1.1501786708831787, -1.4538097381591797, 0.5964634418487549, -0.9403727054595947, -0.47260671854019165, 2.6477208137512207, 1.235751986503601, 0.2103758603334427, -0.38540881872177124, 1.0932176113128662, -0.5688467025756836, 0.10018224269151688, -1.2657709121704102, 0.9808087944984436, -0.42778921127319336, 0.8850533366203308, 0.4634717106819153, -1.352202296257019, -0.008200489915907383, -0.43892717361450195, -1.3438587188720703, 0.5099247694015503, 1.3740994930267334, -0.40282177925109863, 1.6395204067230225, 0.5136550068855286, 1.635562777519226, -1.44670832157135, 0.7407342195510864, -1.2789559364318848, 0.22640538215637207, 1.2403477430343628, -1.6582342386245728, 2.7903597354888916, -1.0364954471588135, 0.6981875896453857, -0.4375402331352234, 0.037548717111349106], [-0.4057719111442566, -0.8082281947135925, -0.13344405591487885, -1.6092766523361206, 1.8550947904586792, 0.20921453833580017, 0.7471847534179688, -0.41525042057037354, 0.5856802463531494, 0.7082648873329163, 1.7673028707504272, 1.043745994567871, 0.6324877142906189, 0.5757213830947876, 1.7430164813995361, -0.556189239025116, -0.39156267046928406, 0.8546910881996155, 0.18126514554023743, 0.15955105423927307, -0.9871118068695068, -0.42420125007629395, 2.1902337074279785, -0.9748014211654663, 0.19337807595729828, 1.04241144657135, 0.2378588169813156, 1.124860167503357, -0.790926456451416, 0.6464040875434875, -0.7224088907241821, 1.111832857131958, -0.013118946924805641, -0.3525055944919586, -2.0315444469451904, -0.27521148324012756, -1.1261287927627563, -0.8167089819908142, 0.23992589116096497, 0.14725682139396667, 2.0116896629333496, -0.3418722450733185, 1.340532660484314, -1.564734697341919, -0.5331697463989258, -0.6193276643753052, -0.20296171307563782, -0.5894034504890442, 1.7280150651931763, 0.15950602293014526, -2.0473387241363525, -0.42086291313171387, -1.4669671058654785, -0.5630830526351929, -1.597876787185669, 0.7868784070014954, -0.31251829862594604, -0.3519337773323059, -0.17609445750713348, 1.350873351097107, -0.3516387641429901, -0.06223282963037491, -0.15591083467006683, 0.5812435150146484, 2.0610926151275635, 0.939587414264679, 0.8248903155326843, 0.05416315421462059, -1.1000487804412842, 0.5981544256210327, 1.6872600317001343, 0.691551148891449, 0.3698384463787079, -0.6919203996658325, -0.4352092444896698, 0.7891544699668884, 0.39715641736984253, 0.567144513130188, 0.43577608466148376, -0.3403491675853729, -0.6956681609153748, 0.2807314097881317, -0.7600475549697876, 1.1195777654647827, -1.4576953649520874, 0.4408356845378876, 0.779253363609314, 2.5271999835968018, 0.14021144807338715, 0.613133430480957, -1.389536738395691, -1.3465900421142578, -0.40463462471961975, -0.75676029920578, 0.551449716091156, 0.509145200252533, -2.1085164546966553, -0.05150644853711128, -0.6825059652328491, 2.9419314861297607, -3.421210289001465, -0.13083693385124207, 1.153159260749817, 1.640886664390564, -0.7097157835960388, 0.08443823456764221, -1.568074107170105, -0.5057312250137329, -1.5770151615142822, 1.000779390335083, 1.1632485389709473, -0.3067438006401062, -1.123037576675415, -0.38526326417922974, 0.03828701004385948, 0.37230828404426575, -1.2455799579620361, -0.9094690680503845, -0.5941553711891174, -0.07621356844902039, 1.1233224868774414, -0.5877768397331238, 1.2292243242263794, -1.5417280197143555, 0.3656196892261505, -1.310785174369812, -0.18908697366714478, 0.2903768718242645], [-1.18159019947052, -2.853602409362793, -0.7469052076339722, 1.4311574697494507, 1.3751434087753296, -0.47270187735557556, 1.2350937128067017, 0.6534584164619446, -1.1128590106964111, -0.741433322429657, 0.7043090462684631, 2.200263261795044, 0.5501486659049988, -1.2862801551818848, -1.319115161895752, 0.7839195132255554, -0.0032796263694763184, 0.16759634017944336, -0.8511933088302612, -0.3909450173377991, -0.49161428213119507, -0.7750004529953003, 0.23747345805168152, 0.5725363492965698, -0.9909383058547974, 0.426665723323822, -1.6215015649795532, 2.6740176677703857, 0.17081736028194427, -1.0643584728240967, 1.0306731462478638, 0.5720414519309998, -0.6967483758926392, 1.9717227220535278, 0.2796141803264618, 1.2908928394317627, 1.5685555934906006, 0.9822209477424622, -0.12485779821872711, 0.3763085603713989, -2.182077169418335, 0.3206270933151245, 0.3940635025501251, -1.5907578468322754, -1.7852169275283813, -0.14625482261180878, -1.329240083694458, -1.3394041061401367, 0.49200475215911865, 0.231955423951149, -1.1079797744750977, 0.6760997176170349, 0.09779257327318192, 0.697921097278595, 0.22400449216365814, 1.4185093641281128, 0.3343333303928375, 0.3704836070537567, 1.1858121156692505, -0.31541967391967773, -0.11195559799671173, 1.3093900680541992, 0.6644707918167114, 1.383278250694275, 0.26103368401527405, 1.5897512435913086, 0.14284385740756989, -0.02681814320385456, 1.5639448165893555, 0.27597373723983765, -0.2300683557987213, 0.03348264843225479, 0.0074106487445533276, 3.0078070163726807, -0.8754600882530212, 0.29231688380241394, 1.0271838903427124, -0.4050833284854889, 0.06998774409294128, -0.12271136045455933, -0.3770434558391571, 2.6161675453186035, 0.10556880384683609, -0.06141689792275429, -0.5654107928276062, 2.141411066055298, 0.04263293370604515, 1.181694746017456, 1.6473586559295654, -1.4971590042114258, 0.17493829131126404, 0.8048452734947205, -0.15181086957454681, 0.50437331199646, -1.1282048225402832, -0.6175025105476379, 0.08997227996587753, 1.4144999980926514, 1.4885555505752563, 1.344636082649231, -0.5567273497581482, -0.40410715341567993, -0.043684590607881546, -0.2395247519016266, 0.3576992452144623, -0.6612774133682251, -0.8724850416183472, 0.18931902945041656, 0.6508265733718872, -1.5406795740127563, -0.4064224362373352, -0.48002713918685913, -0.14644566178321838, -0.3633583188056946, -0.8733577132225037, 1.2775671482086182, -0.5274270176887512, -0.6653589606285095, 0.3303438425064087, 0.32175156474113464, -0.019399750977754593, -1.1991163492202759, 0.6504170894622803, -0.6171306371688843, -0.41756951808929443, -0.1159808337688446, -0.9480528235435486, -0.37844595313072205], [1.4261125326156616, 1.7454417943954468, -1.3706740140914917, 1.5155543088912964, 0.635966956615448, 0.7158016562461853, -0.25127875804901123, 0.20131926238536835, -0.3890705704689026, 1.6741679906845093, -0.43127334117889404, 0.6550419330596924, -1.9013344049453735, 0.9359464645385742, -1.0078434944152832, 0.014354084618389606, 0.4329327344894409, -0.43334320187568665, 0.40013617277145386, 0.25210633873939514, 1.4439613819122314, -1.259641170501709, 0.03672923520207405, 0.573117733001709, -0.0006891794037073851, 0.14829465746879578, -0.5015779733657837, -1.9932703971862793, 0.09285709261894226, -1.1500924825668335, -1.601558804512024, 1.602212905883789, 0.33732160925865173, -1.065605640411377, -0.5795722603797913, -0.880671501159668, 0.7282811999320984, 0.7335541844367981, -0.7726243734359741, -0.5672714710235596, -0.014007321558892727, 0.12230179458856583, 1.0609550476074219, -0.814302384853363, -0.07995474338531494, -1.0532796382904053, -0.6502231955528259, 0.004274043720215559, 1.263611078262329, 1.568054437637329, -0.8298692107200623, -1.1822125911712646, -0.33404481410980225, -0.3797668218612671, -0.596289336681366, 0.42519083619117737, -0.29354774951934814, 0.0051935999654233456, 0.6367382407188416, -1.8227968215942383, 0.7180628180503845, 0.6833981275558472, -0.3774120509624481, 0.6127173900604248, -2.8403656482696533, -1.0432065725326538, 0.10963939130306244, -0.5921295285224915, 0.28366976976394653, 0.9138676524162292, -0.026041043922305107, 0.45972132682800293, 0.032103728502988815, 1.6865900754928589, -0.32070276141166687, 0.2930689752101898, 1.0042099952697754, 1.6751046180725098, -0.8610281348228455, -0.6464002132415771, 1.552473545074463, 0.5603724718093872, 0.2110278159379959, -0.022145569324493408, -0.32350215315818787, -0.028585245832800865, 1.3418911695480347, -0.46677613258361816, -1.3501763343811035, -1.633386492729187, 0.09407810866832733, 0.49825456738471985, 0.07354868203401566, -0.32549065351486206, 0.3237414062023163, 0.29162219166755676, 0.6569802165031433, -0.4019438326358795, 1.3270310163497925, 0.7156696319580078, 0.15888963639736176, 0.12123317271471024, 0.5689932703971863, 0.951763927936554, 0.22079932689666748, -1.0509759187698364, -0.08782978355884552, -0.3133668899536133, -0.508351743221283, -1.179664134979248, 1.2750333547592163, -0.09595831483602524, 1.4847925901412964, 0.384054571390152, -0.1291496902704239, 0.3346727788448334, -0.6440312266349792, -0.3838837146759033, -1.1225106716156006, 1.0677845478057861, -0.7063047289848328, -0.8070732355117798, 0.6552684307098389, -1.3897426128387451, -1.3195934295654297, -0.8827810287475586, 0.18330204486846924, 0.1441407948732376], [-1.203841209411621, -0.9615594744682312, -0.6520026326179504, 0.12830129265785217, -0.5826858878135681, -1.3538119792938232, -0.9205659627914429, 0.16511651873588562, -0.7260349988937378, -1.5344067811965942, 0.6906364560127258, 1.632609248161316, -0.19939085841178894, -1.5809816122055054, 0.44452205300331116, -0.4022102355957031, -0.3680587112903595, 0.20300306379795074, 1.2211878299713135, 0.48073503375053406, 1.9718506336212158, -0.09605864435434341, 0.726855993270874, -1.3680474758148193, -0.5366902351379395, 0.3796271085739136, 0.8577110767364502, -0.26875782012939453, -0.9458696842193604, -0.3026667833328247, 0.6980898380279541, 0.9698288440704346, 0.20232130587100983, 1.2616503238677979, 0.25328153371810913, 0.3911353647708893, -0.8644524216651917, 0.5762299299240112, -0.14134946465492249, -1.070089340209961, -0.27012738585472107, -0.3268513083457947, -0.17681176960468292, -1.708709716796875, 1.7895526885986328, 0.7037668228149414, 0.2817249894142151, -0.9021976590156555, -0.22605715692043304, -0.6629077196121216, -0.10785244405269623, 1.5997450351715088, -1.8233445882797241, -1.9722942113876343, -2.139370918273926, -0.5719270706176758, -0.5440998673439026, -1.0300573110580444, 2.0113234519958496, 0.8821192979812622, 1.0374627113342285, -0.7211188077926636, -0.7405231595039368, -0.6864811778068542, 0.13203057646751404, 1.896606206893921, -1.1849106550216675, 0.589999794960022, 0.5086396336555481, -0.30426788330078125, 1.0883243083953857, -0.17131631076335907, -1.8712610006332397, 0.20570428669452667, 0.10716738551855087, -1.5699743032455444, -0.04036334529519081, -0.05317481979727745, 0.37893983721733093, 0.24774061143398285, -0.6564105749130249, 1.1458017826080322, -0.9895727634429932, 0.9085968732833862, 0.9054454565048218, -0.6189131140708923, -0.34205862879753113, 0.6005292534828186, 0.08400599658489227, 0.22901420295238495, 0.24541893601417542, -1.316290259361267, -0.10791214555501938, -0.3038225471973419, -1.7265262603759766, 0.13718052208423615, -0.6781437397003174, -0.8964824080467224, -0.43816056847572327, -1.6588343381881714, 1.6431218385696411, -0.32749852538108826, -0.9202348589897156, -0.9089018106460571, -1.5852570533752441, 0.8050102591514587, 0.9570397734642029, 0.41427695751190186, -0.6250059604644775, -0.22187085449695587, -0.01728265918791294, 0.26948443055152893, 2.726728677749634, -1.6671792268753052, -1.197877287864685, 0.21803805232048035, 0.40162357687950134, -0.017259836196899414, 2.603367328643799, 1.1076221466064453, 0.2143556922674179, 1.5811227560043335, -0.08225196599960327, 0.6511576771736145, 1.0923787355422974, -2.089618444442749, 0.916467547416687, 1.3643500804901123], [-0.5583478212356567, -0.3764766454696655, -0.1831696331501007, -0.3577837646007538, 0.4403630793094635, -0.8514299392700195, 0.001309512066654861, -0.6292585730552673, 0.21710555255413055, 1.3992221355438232, -0.7410146594047546, -0.3940129280090332, -0.044207170605659485, -2.101999521255493, -1.396895170211792, 0.5475876927375793, 1.0296659469604492, 0.3880074620246887, 0.2876153886318207, 2.212477922439575, -1.1821690797805786, 0.05335907265543938, -0.09397716820240021, 1.1250827312469482, 0.27764061093330383, -2.7595434188842773, 1.5308527946472168, -0.3816656470298767, 0.026009157299995422, -0.7404240965843201, 0.7607239484786987, 2.433812379837036, -0.7062923312187195, 0.5923939347267151, -0.5901599526405334, -1.395585298538208, 0.23111651837825775, -0.6253597140312195, 0.12790223956108093, 1.093032717704773, 1.547109603881836, -0.693370521068573, -0.3690224885940552, -0.3619321882724762, 0.8285886645317078, 0.4915666878223419, -0.18554061651229858, 1.2533129453659058, 0.3521879017353058, -1.363993525505066, 1.0354013442993164, 0.4815558195114136, -1.6516127586364746, 0.04276955872774124, 0.5177088975906372, 1.8666362762451172, -2.309577226638794, 0.23418086767196655, -0.29072174429893494, -0.8166062831878662, 0.7387089729309082, -0.4307343363761902, -0.5643793344497681, -1.735150933265686, -0.9742665886878967, -0.5235045552253723, -0.19482861459255219, 0.2046196311712265, -0.9475417137145996, -0.2118023931980133, -0.23139801621437073, 1.658279538154602, 0.34089547395706177, -0.13473089039325714, 0.8574475646018982, 1.9771337509155273, -1.3221771717071533, 0.8088873028755188, 2.511798620223999, -1.457708716392517, 0.9243426322937012, 1.6144382953643799, 0.6774885058403015, 1.405737280845642, 1.1926429271697998, 1.3819764852523804, 0.4908897578716278, -0.10357638448476791, -1.8773524761199951, 0.33213499188423157, -0.12185655534267426, 0.584196150302887, 1.2937569618225098, 0.16976918280124664, -1.94513738155365, 1.0512536764144897, -0.860837996006012, -0.8926453590393066, -0.5787778496742249, -0.8830888867378235, -0.1585681140422821, -0.9754261374473572, -0.7235100269317627, -0.49786850810050964, -0.17847499251365662, 0.08516241610050201, 0.829940140247345, 0.15474361181259155, -1.2833857536315918, -0.6693815588951111, 0.11148952692747116, 0.171963632106781, 2.6786324977874756, -0.6052752733230591, -1.0081359148025513, -0.004223305266350508, 0.5582552552223206, 0.024769563227891922, 0.3973788619041443, -0.8664761781692505, 1.0469354391098022, 0.8713521361351013, 1.203743577003479, -1.7871952056884766, 0.5148875713348389, 1.9511547088623047, -0.5968382358551025, 0.3610773980617523], [-0.9879357814788818, -1.3083101511001587, -0.2181479036808014, 0.8850144743919373, -0.4284989833831787, 1.049343466758728, 1.274891972541809, 0.2500622570514679, -0.9301916360855103, 0.14941835403442383, 2.427678108215332, -0.5155106782913208, -1.3082231283187866, 0.5238274931907654, -0.6603793501853943, 0.9992715120315552, 0.17939646542072296, -0.13470785319805145, 0.2139335572719574, -0.12053913623094559, -0.1946890950202942, -0.8455128073692322, -0.15952177345752716, -1.193195104598999, 0.8304196000099182, -0.02476181462407112, -0.5964991450309753, -0.9639010429382324, 0.8092243075370789, 1.4443893432617188, 0.7360881567001343, 0.4548065960407257, 1.286281943321228, -0.46039122343063354, -0.21197891235351562, 2.2165474891662598, -0.036671433597803116, 0.08009392768144608, -0.607224702835083, -0.2573966979980469, -0.06542014330625534, 0.7875857353210449, -0.30889084935188293, -1.385628581047058, 0.447417289018631, -1.2629728317260742, 3.3014018535614014, -0.9075909852981567, 0.8220082521438599, -2.001657485961914, 0.35868582129478455, 0.8379891514778137, -0.18931053578853607, -0.9499925971031189, -0.6725357174873352, 0.6177958846092224, 0.1648203730583191, -0.6927975416183472, -0.2848108112812042, 0.4894425570964813, -0.33540257811546326, 1.1761181354522705, -1.1992565393447876, 0.015562747605144978, 0.0024328110739588737, -1.007872462272644, -1.7654072046279907, 1.0248448848724365, -0.16240178048610687, -2.6040799617767334, 1.2479538917541504, 0.9859370589256287, 1.4710471630096436, -0.2872752249240875, -1.2780725955963135, 0.05254880338907242, -0.5638726949691772, 1.3107109069824219, -0.21127645671367645, -0.44676002860069275, -0.22779487073421478, 0.3752559721469879, -0.5176226496696472, -0.5959867238998413, -0.09933441132307053, -0.12684518098831177, 0.8050965070724487, 0.3439185619354248, -0.19076073169708252, -0.3966260850429535, 2.3421263694763184, 0.36496663093566895, 0.2508023977279663, 0.2598416209220886, -0.18667197227478027, -1.1583195924758911, 1.6919159889221191, 0.7795714139938354, -0.017880016937851906, 0.20464298129081726, -0.31193971633911133, -1.2890028953552246, 1.4561755657196045, 1.3416035175323486, -0.14845053851604462, -1.1398813724517822, -1.1252280473709106, 0.3992666006088257, -1.7393697500228882, -0.43769440054893494, -1.03638756275177, -2.5752015113830566, 1.7505744695663452, -0.7334032654762268, -1.250651478767395, -0.1746373176574707, -1.6706453561782837, -1.3019239902496338, -0.32894548773765564, -0.030936021357774734, 0.5486543774604797, -1.3997530937194824, 0.13712434470653534, -0.9055563807487488, -1.0639679431915283, 1.5072920322418213, 2.3164217472076416, -0.6829478144645691], [-0.36354517936706543, -0.42636236548423767, 1.8407022953033447, 0.6253520250320435, 1.3734562397003174, -0.49459171295166016, -0.46132993698120117, -0.21587060391902924, -2.3643124103546143, 0.20469418168067932, 1.1940842866897583, -0.7177186012268066, 1.7171127796173096, -0.5898347496986389, -0.4438422918319702, 0.16666606068611145, -0.6832675337791443, 1.469604253768921, -1.2542685270309448, 2.052581787109375, -2.175313711166382, 1.3563904762268066, -0.575132429599762, -0.29354003071784973, -0.27142658829689026, 1.2434206008911133, 0.5584502220153809, -0.5891945958137512, -1.5158050060272217, -1.7555763721466064, -0.9678031802177429, -1.2212163209915161, 1.3664772510528564, -0.7199677228927612, 0.030525555834174156, 1.2591156959533691, 1.5083115100860596, -0.040498752146959305, -0.783320963382721, -1.249678134918213, 0.05415751785039902, 0.6855045557022095, 1.0550671815872192, 0.4740690588951111, 1.0984997749328613, 0.45635902881622314, -1.4818750619888306, -0.5898245573043823, -1.6145223379135132, -0.570220947265625, 1.4990662336349487, -2.4305801391601562, 0.7069942355155945, -0.9565842747688293, -0.980796217918396, -0.008151233196258545, -2.202329397201538, 2.5231730937957764, -0.9635593891143799, -1.0791015625, -0.14640700817108154, 0.28340718150138855, 1.5181083679199219, 1.0196044445037842, 1.7015838623046875, 1.1072578430175781, -0.07190714031457901, 0.4681924283504486, -1.4035643339157104, -0.6419528126716614, -0.09597595036029816, 0.5906816720962524, -0.7247359752655029, -0.27468177676200867, -1.252814531326294, 0.3637644946575165, -0.9848779439926147, 0.0082432571798563, 0.9238511919975281, 0.06722872704267502, -0.14608411490917206, 0.1104661300778389, 1.3562692403793335, 0.23696522414684296, -0.220514178276062, 0.28172093629837036, 0.9030481576919556, 0.015304495580494404, -0.6168174147605896, 1.5790914297103882, 2.1506423950195312, 0.7991593480110168, -0.7343152761459351, -0.8249358534812927, 0.8429369330406189, -1.1601881980895996, -0.8138615489006042, -2.4207773208618164, -0.24461710453033447, -0.6857385635375977, -1.6572471857070923, -1.14093017578125, 1.3797893524169922, 0.17194126546382904, -0.674067497253418, -0.3230170011520386, -0.4006689190864563, 0.18767254054546356, -1.1803182363510132, -1.0635894536972046, 0.2910084128379822, -0.36550161242485046, -0.018168477341532707, 0.3001783490180969, -1.4280441999435425, -0.5500892996788025, -2.2713353633880615, -1.0271416902542114, 2.9893176555633545, -0.8124693036079407, -1.3910958766937256, -1.0395585298538208, -0.2196832150220871, 0.8809557557106018, 0.9259768724441528, -1.4146511554718018, 0.15749870240688324, 0.5067793130874634], [-1.4418742656707764, -1.4326432943344116, -0.2670764625072479, -0.6813977360725403, -1.0599290132522583, -1.4111888408660889, 0.19555750489234924, -0.63880854845047, 0.6340078115463257, 0.31988900899887085, 0.49249956011772156, -1.7299168109893799, 0.20353543758392334, -2.392094850540161, -1.328077793121338, 1.3546788692474365, -0.09046228975057602, 1.343632459640503, -0.16212423145771027, -1.1194167137145996, 0.31108012795448303, 1.3150371313095093, -0.7095207571983337, -1.4277135133743286, -1.0141644477844238, 0.1513597071170807, 0.38753989338874817, -0.6029528975486755, 1.0900745391845703, 0.8058250546455383, -0.3090236783027649, 0.573745846748352, -1.8154715299606323, -0.5915735363960266, 2.2032933235168457, -0.17363330721855164, 0.2464333325624466, 0.4586256444454193, -0.7347885370254517, -0.8233640789985657, -0.6137421131134033, 0.15730608999729156, -0.832500696182251, 1.2060601711273193, 0.280674546957016, 1.3596892356872559, 1.0993980169296265, -0.5766633749008179, 0.7443957328796387, -0.28227144479751587, -2.0726943016052246, 1.326528549194336, 0.26858359575271606, -0.5893363952636719, -1.8921618461608887, -0.7155972123146057, -0.07019870728254318, 0.1692093163728714, 1.3218634128570557, 0.4712490439414978, 1.2968635559082031, 1.7202658653259277, -1.5995090007781982, -0.4820559620857239, -1.0967496633529663, -0.34529614448547363, 0.40322327613830566, 0.24892671406269073, -0.6497138142585754, -2.1264548301696777, -1.4537031650543213, -0.4342498779296875, 0.23191401362419128, 0.46423473954200745, -0.4760695695877075, 1.5961930751800537, -0.6179954409599304, -0.9012351036071777, 0.02312372624874115, 0.25135353207588196, -0.36452966928482056, -1.288495659828186, -0.8358887434005737, 0.9677667617797852, -1.2435574531555176, -1.1420522928237915, 1.8463480472564697, -1.1128134727478027, 0.14165519177913666, -0.3292084038257599, -0.651656448841095, -1.3986705541610718, 1.4625552892684937, 0.1262672245502472, -1.0224132537841797, -1.337506651878357, 0.5388515591621399, 0.8194653987884521, 0.8748072981834412, -0.6184486746788025, 0.43054160475730896, -0.6354140043258667, -0.46711841225624084, -0.07934312522411346, 0.776913583278656, 0.37796667218208313, -1.4350969791412354, 1.7337415218353271, -0.08508981019258499, -0.009492564015090466, -0.020116999745368958, -0.3210873007774353, 1.057018518447876, 1.7522342205047607, -1.3254448175430298, 0.4336818754673004, -1.986126184463501, -0.4454556405544281, 1.9305336475372314, 0.8823354244232178, 0.722338080406189, -0.5359272360801697, 0.9253324866294861, 0.7570504546165466, 0.38544654846191406, -0.019510285928845406, -0.26879259943962097, -0.1642908751964569], [0.31924375891685486, -0.3946318030357361, -0.9567919969558716, -1.0395877361297607, -0.9880447387695312, -1.6156560182571411, -0.9064640998840332, 0.8438103795051575, -0.5145843029022217, -0.663173258304596, 1.4007444381713867, -0.9003262519836426, 0.19393126666545868, -1.1352579593658447, 0.7548238635063171, -0.09069492667913437, 0.5556057095527649, 2.008524179458618, -0.03571777790784836, -0.5675575733184814, -0.24444343149662018, 0.3449516296386719, -0.4074389636516571, 2.0199193954467773, -1.5113414525985718, 1.0237205028533936, 0.3777715563774109, 1.6044200658798218, 0.017113901674747467, -1.59580397605896, 1.020626425743103, 0.07746929675340652, -0.20113617181777954, 0.20690011978149414, 0.5949859619140625, -0.4019688367843628, 0.947669506072998, -0.19861288368701935, -0.3857319951057434, 0.4221150279045105, -1.9390621185302734, 0.02834896184504032, 0.7376951575279236, 0.9925742745399475, -1.4110069274902344, -1.0579365491867065, -0.6043273210525513, -2.011644124984741, 0.09128021448850632, -0.7210568189620972, -0.22811205685138702, -1.1750588417053223, 0.8407009243965149, 0.10068472474813461, 1.3685009479522705, -0.21258297562599182, -0.9442145824432373, 0.5409788489341736, -0.38102665543556213, -2.010963201522827, -2.2878048419952393, -0.3182578980922699, -1.666313886642456, -1.755649209022522, 0.688110888004303, -0.3490797281265259, -0.7367932200431824, -0.9363977909088135, 1.3215056657791138, -0.7766973376274109, -0.5327622294425964, -0.2859059274196625, -0.4329387843608856, -1.8529653549194336, 0.31515395641326904, 0.6929103136062622, 0.2894611358642578, 0.07185903191566467, -1.0312286615371704, -1.359840750694275, 0.7108418941497803, -0.49791577458381653, -1.0090205669403076, -1.033921718597412, -2.3268070220947266, -0.41069766879081726, 0.4723654091358185, 0.9268057942390442, 0.6767187118530273, 1.184770107269287, 0.17491376399993896, 1.4319838285446167, 0.8660476207733154, -0.9345175623893738, -1.5361742973327637, -1.1148462295532227, 0.17665137350559235, -0.3695240914821625, 1.4212934970855713, -0.7927997708320618, -0.19124872982501984, 0.10537541657686234, -0.3586752712726593, -1.1081535816192627, -0.7199501991271973, -2.5773403644561768, 0.48293378949165344, 0.49883389472961426, 0.48206406831741333, -0.2855207920074463, 0.5459101796150208, 0.5032342076301575, 0.6946613788604736, 0.706355094909668, 0.7082801461219788, 0.5585169792175293, 0.5286029577255249, -0.28618761897087097, -0.3913542330265045, -0.8156011700630188, 0.7336044311523438, -1.972311019897461, -0.12003937363624573, 1.0651509761810303, 0.44645562767982483, -0.1815367192029953, -0.1375264823436737, -1.5196876525878906], [1.113394021987915, -0.15668687224388123, 0.8211494088172913, 1.6941863298416138, 0.2699182331562042, -1.2501190900802612, 1.0725575685501099, 0.34230899810791016, -0.08221208304166794, 1.1337276697158813, -0.12221574038267136, -0.5583428144454956, -0.16786673665046692, 0.5755553245544434, -0.5625889301300049, -1.0853300094604492, -0.32619673013687134, 0.6458284258842468, 0.16308575868606567, -1.0985252857208252, 1.5697755813598633, -0.3845612406730652, -0.13569386303424835, -0.8648427128791809, -0.8469094038009644, -0.19829417765140533, -0.8085392117500305, 1.3264920711517334, 2.280278444290161, 0.24368558824062347, 1.5149219036102295, -0.5814591646194458, 1.817274808883667, -1.649616003036499, 1.9727749824523926, 0.5084307193756104, -0.8265900015830994, -1.197031021118164, 0.2965362071990967, -0.8375332355499268, 0.036106690764427185, 1.3941246271133423, 0.833008348941803, -0.603984534740448, 0.45505374670028687, 0.10307557880878448, 0.17007021605968475, 0.9366786479949951, -0.2924586534500122, 0.39724355936050415, 1.0422883033752441, 0.30269187688827515, 1.2838200330734253, -0.3456633687019348, -0.34571248292922974, 0.37172624468803406, -0.15084956586360931, -2.6151392459869385, 0.3018626868724823, -1.780403733253479, -0.06569700688123703, 0.2856350839138031, 0.8990292549133301, -1.372428297996521, 2.108707904815674, 1.0307903289794922, -1.1784839630126953, 0.32584646344184875, 1.2562693357467651, -0.9066652655601501, 1.2816886901855469, 1.683958888053894, -0.3981189429759979, -0.3215307891368866, 0.8496707081794739, -2.0106019973754883, 1.169026494026184, 0.3640243113040924, 1.783408284187317, 0.6481401920318604, -0.6049346923828125, 1.6509751081466675, 0.2982529401779175, -0.029537376016378403, 0.09209351241588593, -1.2478325366973877, 0.5529107451438904, 1.1992294788360596, 1.2891217470169067, -1.6832021474838257, 0.5241919755935669, -0.037807442247867584, -0.40033844113349915, 0.05186890438199043, 1.3736625909805298, 1.046244502067566, 1.5436354875564575, 0.10144168883562088, 2.122527599334717, -0.6312229633331299, 1.1674922704696655, -0.7247506976127625, 1.4111074209213257, 0.4689164459705353, -0.19181346893310547, 0.11025510728359222, 0.9136728644371033, 0.29515260457992554, -0.12838424742221832, -0.4242539405822754, 1.7329871654510498, 0.6641104817390442, 1.1375555992126465, -0.47797465324401855, -0.6729905009269714, 0.09194138646125793, 0.13892774283885956, 0.3016318380832672, -0.6755359768867493, 0.654765248298645, -0.7429016828536987, 0.6440108418464661, 1.086368203163147, 0.6410067081451416, 0.13629092276096344, -1.4388121366500854, 0.05533469468355179, 1.7924786806106567], [0.4652157127857208, -0.06599415093660355, -0.688768208026886, 0.5465496778488159, 0.8441744446754456, 0.08077896386384964, -1.1611618995666504, -0.5700668692588806, -1.3038029670715332, 0.8922000527381897, 1.104666829109192, 1.5965064764022827, 1.2776838541030884, 1.539791226387024, 0.8563342690467834, -1.355079174041748, -0.23497061431407928, -0.5675452351570129, -1.0054084062576294, 0.11823000013828278, 0.034974757581949234, -0.6692190170288086, 0.4654533863067627, 0.3167324960231781, 0.3669201135635376, -0.7195764780044556, 2.305133104324341, 2.10642671585083, 2.5500102043151855, 1.9741917848587036, -3.1721537113189697, -2.2183926105499268, -0.05914114788174629, -0.6215877532958984, -0.8186403512954712, 0.3567880690097809, -0.5721302032470703, -1.6568207740783691, -0.8979910612106323, -0.5857870578765869, 0.3065209686756134, 0.5153859853744507, 1.2232961654663086, -0.7137459516525269, -0.7345934510231018, 0.9566176533699036, 0.7062274813652039, 1.048339605331421, 1.3811328411102295, 0.5637362003326416, -1.8866095542907715, -2.406423330307007, -0.0164746455848217, 0.8210248351097107, -1.144970417022705, -1.2217721939086914, 1.021982192993164, -0.298806369304657, -0.9992858171463013, 0.9655370116233826, -0.19540441036224365, 0.7644659876823425, -0.08339736610651016, 0.16075368225574493, -0.08649852126836777, 1.1160228252410889, 0.5864575505256653, 0.6044512987136841, -0.24430891871452332, 0.31872257590293884, 0.0031518652103841305, 1.1282740831375122, -0.5144751071929932, 0.8557822704315186, 0.8844049572944641, -0.15766045451164246, 0.6214897036552429, -0.07985437661409378, 1.1353596448898315, 1.0571469068527222, 0.12373004108667374, 0.17346420884132385, 0.5338905453681946, 1.207816481590271, 2.9653987884521484, -0.2810854911804199, -1.1806706190109253, -0.32562023401260376, 0.8903042674064636, 1.2158989906311035, -0.2099969983100891, -0.2847254276275635, -0.2726063132286072, 2.317610502243042, -1.327785849571228, 0.7077056169509888, 0.09287688136100769, -0.14576974511146545, -0.0606597438454628, -0.9582285284996033, -0.2975309491157532, -0.03373271971940994, -0.37183231115341187, 0.9573272466659546, 0.2691915035247803, 1.7227216958999634, 0.8241528272628784, -2.1364364624023438, -1.1119420528411865, -1.122300386428833, 0.1811058521270752, 0.006600061897188425, 0.5177435278892517, 2.010155439376831, -0.14043943583965302, -0.1657623052597046, -1.7537933588027954, 0.6791484951972961, -1.363244891166687, -1.116864562034607, -1.1186145544052124, 1.3698186874389648, 1.5373828411102295, -2.2047510147094727, 1.4760023355484009, -0.10217135399580002, 0.5951071381568909, 0.5937663316726685], [-0.9606337547302246, 0.39646634459495544, 1.392099142074585, -0.7164736986160278, -0.9032056927680969, -0.5887901782989502, -0.04700399562716484, -1.7863448858261108, -0.6469555497169495, 0.30994200706481934, -0.7847321629524231, -0.22327542304992676, 2.0995795726776123, -1.8374735116958618, 1.6580389738082886, -0.8874431848526001, -0.5468068718910217, -0.7425311803817749, 0.519568920135498, 0.21740968525409698, 1.0636446475982666, -0.22002586722373962, -0.1503538340330124, 1.1660584211349487, 2.202714443206787, 0.7759332656860352, -1.1638429164886475, -0.5719519257545471, -1.162987232208252, -0.7475504875183105, -0.2822033166885376, 0.28632599115371704, 1.0209133625030518, -1.2113335132598877, 2.012735366821289, -0.5500588417053223, -0.4183080792427063, -1.5193190574645996, 0.07862614095211029, -0.3694436252117157, 0.10379254817962646, -0.8187404870986938, 1.4140783548355103, 0.4453018307685852, 1.519228219985962, 0.6613616943359375, 0.5799298286437988, 0.5122302770614624, 0.8885661959648132, -0.5717283487319946, 0.607002854347229, -1.110190749168396, 0.24919646978378296, -2.0111937522888184, 0.724759042263031, 0.3000882565975189, 1.1465922594070435, 1.1483477354049683, 0.02759232372045517, 0.6176049709320068, 1.6826452016830444, -0.987928032875061, -0.12485846132040024, -1.6627998352050781, 1.2373865842819214, 1.3286138772964478, -0.6363837122917175, -0.055398814380168915, 1.118179202079773, -1.1233816146850586, 1.0782777070999146, -0.22548642754554749, -0.0481816790997982, -0.4306533932685852, 1.1961712837219238, -0.25365883111953735, 1.2954189777374268, 0.04325040429830551, 1.2743549346923828, -0.25777164101600647, 0.41089779138565063, -0.26600944995880127, 1.024741530418396, -0.334945946931839, -0.9809252023696899, 0.11530555784702301, -0.793899416923523, -0.39543622732162476, 0.32456669211387634, 2.731048822402954, 1.368802785873413, 0.6440547108650208, 3.5973026752471924, -0.9658355712890625, 0.05118059366941452, -1.000172734260559, 0.09354628622531891, -0.11604322493076324, -1.1915595531463623, -0.5324242115020752, -0.27034419775009155, -0.10332654416561127, 1.2702082395553589, 0.18938668072223663, 1.4139982461929321, 1.2233904600143433, -1.4945435523986816, 0.17475582659244537, 0.1138380616903305, -1.1523072719573975, 0.8364748358726501, -1.0975873470306396, 0.11223554611206055, 1.1261216402053833, -0.44949907064437866, 1.2141096591949463, 1.8519906997680664, -0.5246186852455139, 0.7254968881607056, 1.3650349378585815, -1.1449872255325317, 0.42835721373558044, -0.8090125322341919, 0.07605069130659103, -0.9335256814956665, 1.1094757318496704, -1.753691554069519, -0.5580357909202576], [-0.44813257455825806, -0.13002561032772064, -1.1796488761901855, 1.7040197849273682, 2.5740292072296143, -0.3628362715244293, -1.4873511791229248, -0.8949307203292847, -0.7389956712722778, 1.1064316034317017, 0.3322887122631073, -0.6022297739982605, -1.701270580291748, 0.027715880423784256, 0.688528835773468, -0.2052670121192932, 0.9411601424217224, 1.0349160432815552, 0.8793054819107056, 2.2431445121765137, -0.4461342990398407, 0.17127352952957153, -1.3146735429763794, 1.1539862155914307, -0.37221473455429077, -0.7472473978996277, -0.15494544804096222, 1.4027085304260254, -1.0427416563034058, -0.20948779582977295, -0.9828336238861084, 1.214803695678711, 0.5736692547798157, -0.27333322167396545, -1.1368331909179688, -0.5789265632629395, 2.464247703552246, -0.308917760848999, 2.039888381958008, -0.18614642322063446, 0.8874130845069885, 3.515399694442749, 0.7185774445533752, 0.085578553378582, -0.058984313160181046, 0.03289240598678589, -0.3834611475467682, -0.10082446783781052, -1.1675816774368286, 0.5324037671089172, -1.2730445861816406, 0.6092194318771362, -0.32662275433540344, 0.1051558181643486, -0.18396931886672974, 0.23672620952129364, 0.12463736534118652, -0.19111353158950806, 0.58539879322052, -0.9438038468360901, -0.007427255157381296, -0.23562327027320862, -0.5941821336746216, -1.3130055665969849, -1.8171398639678955, 0.5282537341117859, -2.1415798664093018, -0.5443681478500366, 0.3460162878036499, -0.693494975566864, 1.661835789680481, 0.20139609277248383, -0.8271805047988892, -0.3655720353126526, -1.1470530033111572, -0.8584761619567871, -1.3845231533050537, -0.5442992448806763, -0.27849501371383667, 0.40864524245262146, -0.04024793952703476, -0.21366046369075775, 0.6807454228401184, 0.2132439911365509, -0.4952373802661896, -1.1748322248458862, -0.2604143023490906, -0.36608949303627014, 1.0432394742965698, -0.6579042077064514, -1.1118767261505127, 0.3203701972961426, -1.4511141777038574, -0.8851726055145264, 0.7608875632286072, 0.8805558681488037, 0.7685452103614807, 0.6313405632972717, 0.1702609807252884, 0.5171118378639221, -1.2729439735412598, -0.4947929084300995, 1.3174289464950562, -1.3955464363098145, -0.949017345905304, -1.4421851634979248, 2.2976508140563965, 0.14177045226097107, 0.47120776772499084, 0.19742408394813538, -0.8379572629928589, 1.1355408430099487, -1.4792535305023193, -0.7485025525093079, 1.1445845365524292, -0.31613215804100037, 0.00010693016520235687, 2.8650193214416504, 0.2305491417646408, 1.086990237236023, 0.6605414152145386, 1.8392454385757446, 0.7274016737937927, 0.529229462146759, 1.5995762348175049, 0.8013862371444702, 0.18821898102760315, 0.4867947995662689], [1.2610589265823364, 1.1040867567062378, 1.3513069152832031, 0.08083212375640869, 0.1430976390838623, -0.5596779584884644, -1.1331017017364502, 0.5892532467842102, -0.45365461707115173, -0.023006431758403778, 0.9832850098609924, 1.7434196472167969, 0.5393661260604858, -0.4461016058921814, -1.1306382417678833, -0.3928239941596985, 0.6514911651611328, 0.19703710079193115, 0.7293038368225098, 0.38075172901153564, -0.048145636916160583, -1.070529818534851, 1.0918147563934326, -1.744274616241455, 1.283645749092102, -0.257683128118515, -1.0528883934020996, -1.534787654876709, 0.9051691293716431, 0.25630709528923035, 0.37279748916625977, -0.9072970747947693, 2.3660645484924316, -1.2130372524261475, 0.4627251625061035, 0.10172715038061142, 1.145349383354187, 0.522473156452179, -0.4871786832809448, 0.4381132423877716, -1.8743778467178345, 0.6644921898841858, -0.2134784758090973, 1.6267322301864624, 0.14014174044132233, 1.0808521509170532, 0.23165346682071686, 0.3282281458377838, 0.008684013038873672, 0.21194282174110413, -0.8773210644721985, 0.2684978246688843, -0.7715852856636047, 1.6094129085540771, 0.653928816318512, 0.08655824512243271, 0.6137881875038147, -1.404618740081787, 0.8346943855285645, -0.04535093531012535, 0.16204635798931122, -0.10188685357570648, 0.8363777995109558, -0.8519189953804016, -0.9213001728057861, 0.1358787715435028, 0.26491159200668335, 0.0383525975048542, -0.33404552936553955, -1.5800628662109375, -0.3924385905265808, -0.6847104430198669, 0.39156508445739746, -0.24923986196517944, -0.4707172214984894, -1.1893680095672607, -0.013300882652401924, -0.44122442603111267, -0.8600959777832031, -0.6736850738525391, 1.2481424808502197, 2.067075252532959, -1.1450635194778442, -0.6233381628990173, -0.7492883801460266, -1.4971725940704346, -0.6009112000465393, 1.6995463371276855, 1.9244846105575562, -0.5819910764694214, 0.5569407939910889, 0.31877610087394714, 1.0952353477478027, -1.500611662864685, -0.14794135093688965, -0.6186646819114685, 0.023701459169387817, 0.252429723739624, -0.7948124408721924, 1.6323732137680054, -0.40749862790107727, 0.5492081642150879, 1.7750481367111206, -1.0299443006515503, -0.09650078415870667, 0.12138460576534271, 0.4905315637588501, 1.6734687089920044, 0.209230437874794, 0.9455612301826477, 0.7263182401657104, -0.49730074405670166, -1.393373966217041, -0.8358367681503296, -0.8541744351387024, 0.5686931610107422, -0.308051735162735, -0.3047560453414917, 1.2013745307922363, -0.027404403313994408, 0.025461668148636818, -0.5451793670654297, 1.377231240272522, 2.2893195152282715, 0.6166407465934753, 0.24158617854118347, -1.3082209825515747, -0.14342433214187622], [0.0886760726571083, 0.14445605874061584, -0.5173537731170654, 0.7088020443916321, 0.9305999279022217, 0.16152283549308777, 1.1037280559539795, 1.12157142162323, -0.589370608329773, 1.446583867073059, -1.8022855520248413, 0.12504161894321442, -1.005899429321289, -0.9875966906547546, -0.5088398456573486, -0.08237573504447937, -0.18672417104244232, 0.3970356583595276, -2.3970091342926025, 0.7484102249145508, 1.917128562927246, -0.71409672498703, 0.9541494250297546, -0.9995611906051636, 0.04461228474974632, -1.5975565910339355, 0.4307488799095154, -0.030641799792647362, 1.6966687440872192, -0.6294474005699158, 0.8802637457847595, 0.054764099419116974, 0.5182934403419495, -1.0857545137405396, 0.30837079882621765, 1.0177428722381592, -0.2797420620918274, 0.09168712049722672, 0.8121315836906433, 0.7076869606971741, -0.040020786225795746, 0.8328248858451843, 1.2409946918487549, -0.5012056827545166, 0.06808163225650787, -0.01105139497667551, -0.11220481246709824, 0.28963878750801086, -0.6597070097923279, -1.203682541847229, 0.14659984409809113, -1.1683369874954224, -0.7411877512931824, -0.1939469277858734, 0.5439397096633911, 0.21221284568309784, 0.5252321362495422, -0.6345597505569458, -1.078973650932312, 0.25034430623054504, 1.1844838857650757, 1.2168147563934326, 0.35125577449798584, 1.145604133605957, -0.05915190652012825, -0.7839270830154419, -0.5524666905403137, 0.33075928688049316, 1.1262937784194946, 1.4039596319198608, 1.5902118682861328, 0.7938192486763, -0.5998477339744568, -0.6860806345939636, -1.233276605606079, -0.616443395614624, -0.6037271618843079, -0.42942193150520325, -0.33977022767066956, 1.188004970550537, 0.9240955710411072, -0.5308518409729004, -2.4934608936309814, 0.49953293800354004, 0.08327615261077881, 0.7428743839263916, -0.19385899603366852, 0.5878033638000488, 1.6118416786193848, 0.11516346037387848, 0.4455083906650543, -1.0876981019973755, 0.5476579666137695, -2.1787655353546143, -0.6472018361091614, -0.07947126030921936, -0.9331670999526978, 1.0334434509277344, 0.14318948984146118, -0.3795817792415619, -0.6025420427322388, 1.0840784311294556, 0.19972071051597595, 1.424512267112732, -1.2781838178634644, 0.41088759899139404, 1.149430274963379, -0.24366508424282074, -0.07351594418287277, -0.28027546405792236, -0.41109931468963623, 0.04323556646704674, 0.5511549711227417, 1.5987578630447388, 1.6437846422195435, 0.787741482257843, 0.03752943500876427, -1.5119425058364868, -0.9445278644561768, 0.9118102192878723, -0.3743615448474884, 0.35502126812934875, 1.43047034740448, 0.6901648044586182, 1.1978821754455566, -0.6829984784126282, -0.5505804419517517, -0.8383521437644958], [0.8715062737464905, -1.3175400495529175, 1.6370761394500732, 0.5421773791313171, 0.18664735555648804, 1.9916387796401978, -1.022576928138733, 1.0825666189193726, 1.4582583904266357, 0.7968881726264954, -1.9109786748886108, -0.9277921319007874, -0.3030945360660553, 0.7391606569290161, -0.03862468898296356, -0.2547566890716553, -1.1918538808822632, -0.44054916501045227, 0.16635385155677795, 0.734446108341217, -1.726188063621521, -0.3169461190700531, 1.0646183490753174, -0.35742518305778503, -0.2273559421300888, -0.751714825630188, -0.3826776444911957, 0.18552468717098236, 0.13839642703533173, -0.3925158381462097, 1.0963271856307983, -0.9586095213890076, 0.20411869883537292, 0.11568310111761093, -0.324271559715271, -0.3949449956417084, 0.27977466583251953, -1.6956052780151367, 1.6833734512329102, 0.8124067187309265, -0.10195612162351608, 0.8863140940666199, -0.9720550775527954, -2.135305881500244, -0.45736050605773926, -0.18887539207935333, 0.9889172911643982, -0.10975630581378937, 0.7126286029815674, -0.5329652428627014, 1.425537109375, -0.6581724286079407, -1.4538323879241943, 0.7572633028030396, 1.158202886581421, 0.27236899733543396, 0.13145416975021362, 1.6392247676849365, 1.3516243696212769, 0.31634315848350525, 0.2301570624113083, 1.3568586111068726, -0.1232699602842331, -0.600740373134613, -0.06484336405992508, -0.07521750777959824, 1.3608319759368896, -0.825472354888916, 1.1126859188079834, 0.8112237453460693, 0.6792923212051392, 0.8991660475730896, 0.05075932294130325, -1.1129647493362427, -0.2826981842517853, -0.8732864260673523, 2.207052230834961, 0.8264896273612976, 0.004233148880302906, -0.782017171382904, -1.8772332668304443, 0.4524040222167969, 0.13598622381687164, -0.9197794198989868, -0.48429080843925476, 1.4300512075424194, 1.0955396890640259, 1.7627559900283813, -1.9364577531814575, 0.850395679473877, 0.281414270401001, 0.566749632358551, 1.2259427309036255, 0.9698328375816345, -0.6877927184104919, 1.9315193891525269, 0.09927931427955627, -0.4493911564350128, -0.017588220536708832, 0.7056578397750854, -0.07078393548727036, 1.2136907577514648, 1.7399022579193115, -1.050625205039978, 0.6847200989723206, 0.16308905184268951, 0.7616716027259827, -1.085155963897705, -0.6609418392181396, 1.047961711883545, -1.0586408376693726, -0.4579004943370819, 0.4034178555011749, 1.0093237161636353, -1.72940993309021, -0.7884783744812012, -0.4386357367038727, 0.11994282901287079, -0.006877140142023563, -0.20199517905712128, 0.4608834385871887, 0.424278199672699, 0.4049798846244812, -0.36043545603752136, 0.8132514357566833, 1.5893092155456543, -0.354696124792099, -1.3396048545837402], [1.820168375968933, -0.0707668736577034, -0.25467851758003235, 0.4024721086025238, -2.4004437923431396, 1.3540302515029907, -0.691010057926178, -0.010712161660194397, -0.1769842952489853, -0.2070886492729187, 0.718636691570282, 0.7977895736694336, 3.0227911472320557, 0.5511413216590881, -0.10710360109806061, 0.04484002664685249, 0.1906156837940216, -1.6320772171020508, 0.3917865455150604, 0.17987750470638275, 0.3374136984348297, -1.0136135816574097, -0.3104332387447357, 1.7297990322113037, -0.3206518888473511, 1.0080963373184204, 0.8290652632713318, 1.6849536895751953, -0.27257877588272095, 0.460332453250885, -0.27738118171691895, -1.6368098258972168, 0.9057375192642212, -0.9826779365539551, -0.49246349930763245, 1.3855931758880615, 1.7365810871124268, -0.7705590128898621, 0.24344958364963531, -2.233228921890259, 0.2601473927497864, -0.8957046866416931, 0.3490714728832245, 0.21534204483032227, 0.5174620151519775, -0.25036191940307617, 0.058466095477342606, 0.01818733662366867, 1.5756468772888184, 0.1195433959364891, -0.9917546510696411, -1.0851433277130127, 0.11223248392343521, 0.9349345564842224, -0.06281055510044098, -0.7593872547149658, 0.05916199833154678, 1.1255784034729004, 0.5911614298820496, -0.5321545600891113, -0.014423223212361336, -0.6762651801109314, 0.31475353240966797, -0.6777457594871521, 0.2084219604730606, -0.9367153644561768, 0.05901110917329788, -0.42751941084861755, 0.42705851793289185, 1.1960558891296387, 0.39941084384918213, -0.31121718883514404, 1.4585767984390259, 1.760290503501892, 0.6311680674552917, 0.42828813195228577, -0.5243129134178162, -1.1931637525558472, -1.7373178005218506, -0.6557032465934753, 0.9984699487686157, -1.3509434461593628, 0.21694409847259521, 0.95582115650177, -0.03438267484307289, 0.6287845969200134, 1.3698859214782715, -1.2755669355392456, 0.023594951257109642, -1.3388715982437134, -1.4411728382110596, 0.8888623714447021, -0.3349681496620178, 0.5393149852752686, 0.9141398668289185, -1.6431443691253662, -0.7106870412826538, 1.233816146850586, 1.5586107969284058, -2.3038549423217773, -1.5026532411575317, 0.6860861778259277, -0.03961636498570442, 0.4221007227897644, 0.9818860292434692, 1.916542410850525, -0.7596181631088257, -1.3053674697875977, 0.3930203318595886, 0.6988825798034668, 0.426146537065506, -1.192634105682373, 0.18855993449687958, 0.3321824371814728, -0.2201359122991562, -0.7675288915634155, 0.2933923602104187, -0.9125947952270508, 1.3204575777053833, 1.597546935081482, -1.2830462455749512, -1.5502351522445679, -1.854306936264038, -1.9087834358215332, -0.8109269738197327, 0.03343987837433815, 0.7232097387313843, 2.369797706604004], [1.1376540660858154, -0.6034818887710571, -1.2884302139282227, 0.15406568348407745, 0.6875466108322144, -0.9720901250839233, 1.5042890310287476, -0.7337709069252014, -0.022290684282779694, 0.711497962474823, -0.6867437362670898, -1.225076675415039, 0.3735334277153015, -1.1739424467086792, -0.7795138359069824, 0.6670461893081665, -1.5579020977020264, 1.491290807723999, -0.8441088199615479, 0.39991438388824463, -1.804611325263977, 0.6555398106575012, 0.23030725121498108, -1.3681061267852783, 0.9853777885437012, 1.3543765544891357, -0.4946952760219574, 0.9193693995475769, -0.5248844623565674, 2.1469526290893555, -0.9024722576141357, 0.31816011667251587, -0.32833513617515564, 0.5968291759490967, 0.7828689813613892, -0.03255925327539444, -0.6510273814201355, 0.9690494537353516, 0.4427156150341034, -1.1574835777282715, 3.1499197483062744, 0.6978086829185486, 3.781775951385498, -1.7557156085968018, -0.036147862672805786, -0.9553422927856445, 0.04241032525897026, -2.16715669631958, -0.3553953170776367, 0.7403756976127625, 1.2835661172866821, 0.11749816685914993, -0.25689375400543213, -0.8138211369514465, 0.48333483934402466, 0.4362501800060272, -1.5030807256698608, 0.08689998090267181, 1.6066824197769165, -1.1586463451385498, -1.704430341720581, 1.5411858558654785, 1.2684015035629272, -0.038409046828746796, -0.5393103361129761, 0.8134520649909973, -0.939060389995575, 1.0272973775863647, 1.2271982431411743, -0.44087904691696167, -0.7776392102241516, -1.1036241054534912, -0.25003471970558167, 0.8940643668174744, 0.3703485429286957, -0.7435706257820129, 0.34606459736824036, -0.7029657363891602, 0.37706732749938965, 1.5547163486480713, 0.9072831869125366, 0.46696364879608154, -0.6179125905036926, -2.114412784576416, -0.5963342785835266, 0.04731389880180359, -1.1602507829666138, -0.6698991060256958, -0.24252831935882568, -0.972852349281311, 1.4459521770477295, -0.1825844943523407, -0.5105578303337097, -0.6668713092803955, 0.15001507103443146, 1.2060807943344116, -0.7473549842834473, -0.06368411332368851, 1.1456724405288696, -0.041288018226623535, -2.637052059173584, 0.5647737979888916, -0.4663754403591156, 1.284095287322998, -0.38282445073127747, 1.427431344985962, -0.6901445984840393, 0.18212872743606567, -0.403426855802536, 0.27931490540504456, 0.2510894238948822, 0.7941832542419434, -0.637097179889679, -0.463369220495224, -1.4830422401428223, 0.45855483412742615, 0.24321617186069489, -1.3895145654678345, -1.0839431285858154, -0.7044283747673035, -1.295523762702942, -1.0538326501846313, 0.5206286311149597, 0.8636967539787292, 0.39185985922813416, -0.4531162977218628, -0.030139410868287086, -1.1707019805908203], [-0.18948574364185333, 0.30822113156318665, 0.054810427129268646, 1.2883121967315674, -0.6823484897613525, 1.2800978422164917, -0.04279748350381851, 0.8500811457633972, -0.37972956895828247, -0.11155404895544052, -1.2723745107650757, 0.8831928372383118, 1.2952098846435547, 0.20756137371063232, 0.1500220000743866, 0.395466148853302, 0.2235954850912094, -2.438898801803589, -0.19920167326927185, -0.04543021693825722, -0.1255202740430832, -0.9043362736701965, -1.3668372631072998, 1.2092291116714478, 0.4417630136013031, -0.17494069039821625, -0.4982164800167084, -0.07829615473747253, -0.34879621863365173, 0.6512877941131592, 0.15909580886363983, -1.8927934169769287, -0.9154860973358154, -0.841092050075531, 1.5390944480895996, -1.6585791110992432, 0.701602041721344, -0.3673284351825714, -0.22469528019428253, -1.0694912672042847, -0.7738832831382751, -0.6756041049957275, -0.7270308136940002, -1.7224489450454712, -1.396767497062683, -0.40184301137924194, -0.5827367305755615, -0.01659591682255268, -0.22008900344371796, -0.08560563623905182, -0.30907508730888367, 1.110887050628662, 0.8838918209075928, 0.5656027793884277, -0.026503104716539383, -0.4002346396446228, 0.46530020236968994, -0.38812097907066345, -0.0179912019520998, -0.7569178938865662, 0.8649154305458069, -0.23549588024616241, 0.3102570176124573, 0.9556566476821899, 1.413002848625183, 0.22126269340515137, 0.21891078352928162, 1.5466347932815552, 0.607022225856781, -0.09479586035013199, 1.1270431280136108, -1.57645583152771, 0.924535870552063, 0.7885586619377136, -1.1455694437026978, 0.8001826405525208, -0.7997161746025085, 0.72452312707901, -0.5222387909889221, 0.49850109219551086, -1.0896031856536865, -0.8291086554527283, -0.12561775743961334, 0.3388441503047943, -0.7398838996887207, 1.1532323360443115, 1.1773202419281006, -0.5418291091918945, -1.3317641019821167, 0.35892006754875183, -0.14606837928295135, -0.40827351808547974, -0.10513468831777573, 0.6726971864700317, 0.28134992718696594, -0.5064086318016052, 0.8982909321784973, 0.27907273173332214, 0.5600299835205078, -1.4674164056777954, 0.3395017385482788, 0.3187587261199951, -1.3654946088790894, 0.011819870211184025, -0.39121559262275696, -1.7980157136917114, 0.15512534976005554, -0.05496175214648247, -0.5339733362197876, 0.5603838562965393, -0.1952059268951416, -0.09641309827566147, -0.7755714058876038, 1.4676169157028198, -0.9658129215240479, -1.5384364128112793, 1.182565450668335, 1.328738808631897, -2.1173973083496094, 0.1190357506275177, 0.240342915058136, -2.0092287063598633, -2.050488233566284, -0.29277706146240234, -0.3494794964790344, 0.9267983436584473, -0.240866556763649, 0.12299732863903046], [0.09411745518445969, -1.8039681911468506, -0.26056385040283203, 0.5139756202697754, -0.968822181224823, 0.9975171685218811, 1.062661051750183, -0.6219176650047302, 0.33292070031166077, -0.29755595326423645, -0.5528262853622437, -0.5316406488418579, -0.25182846188545227, -2.3427388668060303, 0.1171228289604187, 1.5124592781066895, 1.2247906923294067, -0.5701493620872498, -0.9747312664985657, -0.26204782724380493, 0.5920752882957458, 0.6082141995429993, -2.2086400985717773, -0.16437794268131256, -0.499415785074234, -0.1457509696483612, 1.5887752771377563, -2.037017345428467, 0.05588950961828232, -0.7578247785568237, 0.6700415015220642, 0.4137488603591919, -0.6719561815261841, 1.4908246994018555, 0.2334296703338623, -0.1968475729227066, 1.533532738685608, -0.44914719462394714, -1.6018940210342407, -2.280806303024292, -0.09961346536874771, 0.17517894506454468, 1.1477018594741821, -0.5966320633888245, -0.7070205211639404, 0.28372862935066223, -1.1100081205368042, -0.05845797434449196, -0.884250819683075, -0.18731705844402313, 0.31423741579055786, 0.40202170610427856, -0.2702200412750244, 0.5941079258918762, 0.4328450560569763, 0.9324436783790588, 0.335870623588562, 0.14230260252952576, 0.7676709890365601, -2.313652276992798, -0.12513454258441925, 1.3788251876831055, -0.04731062799692154, 1.8950849771499634, -0.5041353702545166, -1.1604325771331787, -1.6986274719238281, 1.31460702419281, -0.5628029704093933, 0.18562696874141693, -0.19237075746059418, 1.0930038690567017, -0.11540328711271286, -0.22852784395217896, -1.6732475757598877, 0.0008627046481706202, 0.6366298794746399, -0.6563210487365723, 0.12099812179803848, -0.6856749653816223, 0.48954567313194275, -0.4085141122341156, 0.5777456760406494, -0.1418226957321167, 1.168122410774231, -1.09257972240448, 0.342278391122818, -0.6299504637718201, -0.16708533465862274, 0.7691465616226196, 0.07124242931604385, 1.601225733757019, -0.9407801628112793, 0.482477068901062, -0.44908270239830017, 0.11823786050081253, 1.3146748542785645, -0.6956305503845215, 1.5694725513458252, -0.7574234008789062, -0.07923140376806259, 0.4142260253429413, -0.768262505531311, -0.8977240920066833, -0.4325005114078522, 2.5634050369262695, -1.3218661546707153, 1.587430715560913, -0.4890959560871124, -1.1305851936340332, -0.3408682644367218, -1.2043156623840332, -2.5129568576812744, 0.06429243087768555, 0.5416719913482666, 1.2239423990249634, 1.1164827346801758, -0.507577121257782, -2.1596720218658447, -0.6762502193450928, 1.440150260925293, -1.753095269203186, -1.1185765266418457, -0.5946805477142334, -1.6255829334259033, -0.7570950984954834, 1.1144850254058838, -0.04026593267917633], [0.2447660118341446, 0.46149685978889465, -0.6329536437988281, 0.016282781958580017, -0.5741618871688843, 1.663793683052063, 0.3732549548149109, 1.2900111675262451, 0.4414823651313782, 0.3355304002761841, 1.0846011638641357, 0.39118561148643494, 1.4265680313110352, -0.7462683916091919, -1.255879521369934, 0.6091519594192505, 1.1265870332717896, -1.2613248825073242, 0.3343755304813385, 0.67598956823349, -0.22581076622009277, -0.9352304339408875, 0.3299289643764496, -0.652159571647644, 1.213390588760376, -0.9772459864616394, 1.1326323747634888, 1.9175758361816406, -1.0538325309753418, -0.48536792397499084, 0.546653687953949, 0.32358309626579285, -0.5168946981430054, 0.1429300308227539, -0.6784319281578064, -1.264905571937561, -0.3180784285068512, 0.4587738513946533, -0.474417120218277, -1.7906079292297363, 1.2643296718597412, -0.7729059457778931, -1.2293612957000732, 1.9819409847259521, 1.7368422746658325, 0.21039573848247528, 0.42564859986305237, -0.07123042643070221, -0.14737656712532043, 0.17470835149288177, 0.24471884965896606, 0.7736198902130127, 1.0977498292922974, -0.31954342126846313, -0.8174154162406921, 0.254851371049881, -0.15014128386974335, 1.4376846551895142, 0.625285267829895, -0.7344803810119629, -1.4023194313049316, -1.7066563367843628, -0.9034457802772522, -0.9018018245697021, 0.8881639838218689, 1.2950564622879028, 0.24056456983089447, -1.1363346576690674, 0.18500953912734985, -0.30045104026794434, 1.8223111629486084, -1.4997953176498413, 0.6149839758872986, 0.30297645926475525, -1.1741245985031128, 1.1478029489517212, -0.9413424730300903, 0.4232901334762573, -1.6124229431152344, 0.84578537940979, -1.4744799137115479, -0.0038975835777819157, -0.7647051215171814, -0.03765695542097092, -0.5710165500640869, 1.5024480819702148, -1.5165537595748901, 0.8699204325675964, 0.5272228121757507, -0.7842587232589722, -0.5282149910926819, 2.847433090209961, -0.01349780336022377, 1.568420171737671, -0.45486775040626526, -0.8564525246620178, 0.25411728024482727, -0.18911541998386383, -0.390651673078537, -0.23872818052768707, 0.9469940662384033, 0.6938055753707886, -0.6455655694007874, -0.43607479333877563, 0.2769109308719635, -0.07261822372674942, 0.26332712173461914, -0.46117573976516724, -0.3704869747161865, 1.4907417297363281, 0.38474875688552856, -0.14857080578804016, -0.9429118037223816, 0.38150498270988464, 0.07289273291826248, -1.7642751932144165, -0.5847663879394531, -0.11590786278247833, -1.5001065731048584, 0.5212512016296387, 0.13506673276424408, 1.3968472480773926, 1.176142692565918, -0.526031494140625, -0.06274305284023285, 0.11328642815351486, 0.17628838121891022, 0.7285598516464233], [-0.005983575712889433, 1.4113878011703491, 1.3026248216629028, -0.500929594039917, -1.561394214630127, 0.5884178280830383, -0.322958379983902, 1.3621374368667603, -0.16986852884292603, 0.9105108976364136, -0.997446596622467, 0.7354539036750793, -0.9282095432281494, -1.6127299070358276, 0.4022921025753021, -0.8604150414466858, -0.26902246475219727, -1.5185744762420654, 1.143295168876648, -0.7163010835647583, 1.5993711948394775, 1.3431707620620728, -1.4239795207977295, 0.6959758996963501, 0.5253583788871765, -0.981579601764679, 1.056796669960022, 0.2810480296611786, -1.0208477973937988, -0.25879403948783875, -0.4138047695159912, -0.8727529644966125, 0.649116039276123, -0.6042114496231079, 0.2977234125137329, 0.3929271399974823, 0.3013208508491516, -3.268073081970215, -0.14100857079029083, 0.42125654220581055, -1.1712528467178345, -0.9012458920478821, 1.089680552482605, -1.606480360031128, 0.9138601422309875, -1.1329759359359741, 1.650467038154602, -0.5823144316673279, -0.47989267110824585, 0.3450123965740204, 1.063159704208374, -0.7307308316230774, -1.2490262985229492, -0.8845866918563843, -0.04186048358678818, 1.197682499885559, 2.225651264190674, 0.16056238114833832, 0.0899413526058197, -0.12310449779033661, 0.8642337918281555, -0.4010537266731262, -0.2296198457479477, 2.0688400268554688, -0.6328169703483582, -0.6265005469322205, 0.3348322808742523, 0.6128578782081604, 1.1238338947296143, -0.9096003174781799, 0.9335702061653137, 0.6663300395011902, 0.9757553339004517, 0.912036120891571, -2.6298515796661377, 1.0548526048660278, -0.8280993103981018, -1.079919457435608, 0.3251344859600067, 1.451873540878296, -0.8332589268684387, 0.28009700775146484, -0.34200745820999146, 2.015108585357666, 0.7498697638511658, 0.6835613250732422, 1.6265771389007568, 0.17737580835819244, -2.122440814971924, -0.4784508943557739, -0.19570429623126984, 0.23951533436775208, 0.23938299715518951, -1.0677915811538696, -0.6498832106590271, 0.5250700116157532, -0.454169362783432, 0.4318963587284088, 0.17633487284183502, -0.2935008108615875, -1.4708611965179443, 0.3695472478866577, 0.07208939641714096, -1.7056591510772705, 2.092874765396118, 0.07058345526456833, 0.09973906725645065, -0.168464794754982, -0.2742786705493927, -0.834926187992096, -2.200976610183716, -1.6718018054962158, 0.11635429412126541, -0.6007007360458374, -1.4233779907226562, 0.13440275192260742, -1.6381677389144897, 0.04978662356734276, 0.471962034702301, 1.2774611711502075, -0.9335002899169922, -0.4142555594444275, -1.5046412944793701, -0.39824041724205017, -0.03614027053117752, -1.5325006246566772, 1.6596604585647583, 0.21684758365154266], [-0.746888279914856, 0.7298562526702881, 0.4409979581832886, 0.32240575551986694, -0.5068615674972534, -0.1099507063627243, -0.07523749023675919, -0.2412519007921219, -0.6426459550857544, -0.2876071631908417, -0.031741175800561905, -0.6736345291137695, 0.6395487189292908, -1.363205075263977, 0.8768255710601807, 0.3300009071826935, -1.3371435403823853, -0.6010357737541199, -0.735184371471405, -1.1066561937332153, 1.1969654560089111, -1.0484325885772705, 1.279737114906311, 1.3861967325210571, -0.712529182434082, 0.788497269153595, 2.6479434967041016, 0.2884577810764313, 0.30054277181625366, 0.040359679609537125, 1.0573474168777466, -1.2523562908172607, -0.8901972770690918, -0.8643385171890259, 0.5318257808685303, -2.421281337738037, -1.5041459798812866, 1.8722584247589111, 1.6525204181671143, -0.6830832958221436, -1.1568368673324585, -0.0903732180595398, 1.1590524911880493, -0.2994539439678192, 0.28556567430496216, -0.21245603263378143, -1.0998256206512451, -0.5001055002212524, -0.3855637311935425, -0.062289975583553314, 0.2073240876197815, 2.0462653636932373, 0.37170684337615967, 1.6573338508605957, 0.17888200283050537, -0.26844385266304016, 0.9778140187263489, -0.07905369997024536, 1.5689847469329834, 0.9620531797409058, -0.5693052411079407, 0.5346065163612366, -0.18252870440483093, -0.08803923428058624, -0.9531571865081787, 0.9434134364128113, -0.49843212962150574, 1.5422935485839844, 1.2469931840896606, 0.61323082447052, 0.9304488897323608, -0.663870632648468, -1.5156468152999878, 1.1749107837677002, 1.8997429609298706, -0.941077470779419, -1.451339840888977, -0.33299046754837036, 0.2240966558456421, -0.30898579955101013, -0.7946396470069885, 0.4903898537158966, 0.05021626502275467, 0.5812122821807861, -0.7856221795082092, 0.7481939196586609, -1.156384825706482, 0.13137930631637573, -1.6163418292999268, -0.4969625473022461, -1.799583077430725, -0.19141459465026855, -0.7438542246818542, 1.076704740524292, -0.00678902817890048, 0.9076809287071228, -0.25024059414863586, 0.15363112092018127, -0.08317896723747253, 0.8304197788238525, 0.7692674994468689, 0.05631069839000702, -0.3233530521392822, 2.1239988803863525, -0.8184458017349243, -1.7135189771652222, -0.33635956048965454, 0.5608400702476501, 0.5498939156532288, 0.48662084341049194, -0.8377702832221985, 1.2306474447250366, -0.4146026074886322, 0.3527143597602844, -1.660675287246704, 1.7469191551208496, -1.0882740020751953, -0.8249647617340088, 0.4138681888580322, -0.6666907668113708, -0.8569666147232056, 1.1359670162200928, 0.7708715796470642, -0.4923759996891022, -1.4059081077575684, -0.2521410286426544, 0.9531877040863037, -0.998929500579834], [-0.8127986788749695, -0.36148738861083984, 0.5371900200843811, 0.20098018646240234, -0.33263683319091797, 0.0964946448802948, -1.574344277381897, 0.6015411019325256, -0.7129696607589722, 0.8790431618690491, -1.0876582860946655, 0.7351844906806946, 1.0068422555923462, -2.046780824661255, 0.7031422257423401, 0.1013716608285904, -0.488333135843277, -1.8387757539749146, -0.23421594500541687, 0.5118805170059204, 1.8211220502853394, 1.3325440883636475, -0.41038504242897034, 0.5135353207588196, 0.09970255196094513, -0.5627663135528564, 1.1247978210449219, 0.7332815527915955, -0.2471417784690857, -1.7701244354248047, -0.4719083905220032, -2.197760581970215, 0.6346349716186523, -0.1056683138012886, 0.6745113730430603, -0.5943995714187622, -0.48358821868896484, 0.7805432677268982, 0.48708873987197876, 0.9383949637413025, 1.5160436630249023, -0.23742477595806122, 3.0349199771881104, 1.8527414798736572, 1.1190943717956543, 0.26148924231529236, 0.15658308565616608, -0.3155013918876648, 0.8725534677505493, 0.18668535351753235, -0.7716221213340759, 0.41277390718460083, -1.190142273902893, -0.08158349245786667, 0.41610240936279297, -1.3878344297409058, 0.08636525273323059, -0.8587873578071594, -0.7802782654762268, 0.9993433952331543, -0.261572927236557, 0.37461918592453003, 0.6475968360900879, -0.3107379078865051, 1.0295268297195435, -0.33435797691345215, -2.1279702186584473, 0.31876441836357117, 1.8369981050491333, -0.2150164395570755, 0.8350484371185303, -0.41702985763549805, -0.41518592834472656, -1.0776054859161377, 0.7671626806259155, -1.8446074724197388, -0.4545789957046509, 0.9870574474334717, -0.2844054400920868, -0.08870209008455276, -0.11339333653450012, 0.7613571882247925, -1.6924160718917847, -0.2070159614086151, -0.2328338921070099, 0.610177218914032, 0.3338358998298645, -2.24845027923584, 0.7967739701271057, -0.14951829612255096, 1.3789350986480713, -0.33148667216300964, -0.1798057109117508, -0.6482121348381042, 0.9525177478790283, 0.407888799905777, -0.629298210144043, 0.7073100209236145, 0.2831636071205139, -1.4297393560409546, -1.5126820802688599, -0.6686965227127075, 0.8393918871879578, 0.5706257820129395, -0.5382395386695862, 0.5119798183441162, -0.3612027168273926, 0.062350574880838394, 1.066825270652771, 0.5549873113632202, 0.6618029475212097, 0.4625619649887085, -0.43457114696502686, -0.08568233996629715, 0.4412204623222351, -1.3806196451187134, -0.49950772523880005, -1.0075136423110962, 0.6664385795593262, -1.4418412446975708, 0.41759952902793884, 0.7712258100509644, 0.5829719305038452, -0.4046511948108673, -0.01638942025601864, -1.2932802438735962, 1.6605098247528076, -0.19664549827575684], [0.6421281695365906, -0.021849602460861206, -0.001022180775180459, -0.4333014190196991, -0.9217462539672852, -1.214048147201538, 0.030446801334619522, -0.18715429306030273, -0.43001192808151245, -0.513929545879364, -0.9442569613456726, -1.3738980293273926, -1.0484586954116821, -0.20869742333889008, -1.5167872905731201, 0.41430461406707764, 1.4012378454208374, 0.6871628761291504, 0.6603675484657288, -1.1983414888381958, 0.9192841053009033, 0.13932472467422485, -1.4331196546554565, 0.5926997661590576, 0.2564004361629486, 1.3184102773666382, -0.006875971332192421, 1.654868483543396, -0.2527613341808319, 0.18441496789455414, -0.26495927572250366, 0.022539611905813217, 0.6850605010986328, -0.7985510230064392, -0.8201749324798584, 1.1353679895401, 1.5526472330093384, -0.4526137709617615, -0.324421763420105, 0.24332402646541595, 0.7535991072654724, -0.23630085587501526, 0.5868064761161804, -0.9501832723617554, -0.7913445830345154, 0.37920013070106506, -0.01373518817126751, 1.6362850666046143, -0.5761903524398804, -0.5402950644493103, 0.8465831279754639, 1.3568463325500488, 0.36223167181015015, -0.7272855043411255, -0.7099135518074036, 0.04894379526376724, 0.6045421957969666, -1.5932427644729614, -0.6223369836807251, -0.5815420150756836, 0.8106145858764648, 0.6799525022506714, -1.5674015283584595, 1.5193157196044922, 1.0184640884399414, 0.5882672071456909, -0.8392923474311829, -0.40046027302742004, 0.3514900803565979, -1.0378670692443848, -1.0605653524398804, -0.31615784764289856, 0.2249295711517334, 0.4953032433986664, -0.5108792781829834, -1.0911864042282104, 1.195855975151062, 1.003664493560791, 0.8794687986373901, 1.0618277788162231, -0.4257408082485199, -0.6296576857566833, 1.45114266872406, 0.6911429762840271, 0.940601646900177, 0.0677395835518837, 1.2658536434173584, -0.31687095761299133, 0.8571318984031677, 0.753031313419342, 0.24817128479480743, 1.4058197736740112, 0.18959760665893555, -1.3837742805480957, 0.6726384162902832, -2.5304338932037354, -0.7487031817436218, 0.01881173439323902, 0.4782506227493286, -1.0793378353118896, -0.9447135925292969, 0.614636242389679, 1.434095859527588, 1.71779203414917, -1.4816538095474243, -0.2608443796634674, -1.10720956325531, 0.7119618058204651, 0.18921014666557312, -0.21341557800769806, 0.9638840556144714, 0.9396551847457886, 0.5051775574684143, 0.719861626625061, -0.41441231966018677, -1.064624547958374, 0.09127480536699295, 0.8539327383041382, 0.8231799006462097, -0.3009321391582489, -0.4255560636520386, 1.10575532913208, -0.3355211019515991, 0.17024153470993042, 0.12316272407770157, 0.011648214422166348, 1.1286591291427612, -0.38424578309059143], [-0.15475256741046906, -2.429672956466675, -1.7769451141357422, 1.936871886253357, -0.1061733067035675, 0.4379099905490875, 0.2378869503736496, 0.26302796602249146, -0.03720900043845177, -1.4198591709136963, 0.8120472431182861, 0.7924329042434692, -0.8403787612915039, 1.5996507406234741, 0.2522849142551422, 2.756148338317871, -0.9618062376976013, 0.21897418797016144, 1.0501627922058105, 0.7955169677734375, -0.2112501561641693, 1.104688048362732, 0.6456941366195679, 1.1682482957839966, -2.1231348514556885, 1.3566219806671143, -0.19394440948963165, -1.4119995832443237, -0.5038023591041565, -0.5830456018447876, 0.936646580696106, -0.817937433719635, -0.30624911189079285, -1.5835720300674438, -0.19278401136398315, 0.6747621297836304, -1.5823042392730713, -0.7062907218933105, 0.6429787278175354, -0.020199796184897423, 0.8140385150909424, -0.0298044141381979, 0.0949544906616211, 2.0466227531433105, 0.8747974634170532, 0.46922415494918823, -0.5785823464393616, 0.2936009168624878, 0.7892258167266846, 0.3999788463115692, 0.4563497304916382, -0.7161324620246887, -0.4946957528591156, 0.2986937463283539, 0.627013623714447, -0.629084050655365, -0.9915807247161865, -1.2860286235809326, -0.24806509912014008, 0.36266663670539856, -0.7539509534835815, -0.9159796833992004, 0.5129892826080322, -0.7659558653831482, 0.5141393542289734, -0.11397112160921097, 1.6020647287368774, 0.020716119557619095, -0.90525883436203, 0.22997990250587463, -0.6584793329238892, -1.1671454906463623, -0.4471031129360199, -1.6639723777770996, -0.30215442180633545, -0.5647557973861694, -0.9410176277160645, 1.0226576328277588, -0.6106995344161987, 0.3649923503398895, 0.9947285056114197, 0.03860161826014519, -1.727194905281067, 0.11710890382528305, 0.5540628433227539, -0.6221849918365479, -0.07920794934034348, 0.27891483902931213, -0.6497475504875183, 0.4105985462665558, -1.3172461986541748, -0.5925548076629639, -0.5704255700111389, -1.3550384044647217, -0.653475821018219, 1.5846283435821533, -1.4638584852218628, 0.7433646321296692, 1.8847694396972656, 0.5973578691482544, -0.6509997844696045, 0.9259183406829834, -1.3315685987472534, -0.36907973885536194, -1.9681689739227295, 1.2843749523162842, 1.6721657514572144, -0.4277040362358093, -1.880458950996399, -0.7021753787994385, -0.610638439655304, -0.4559808671474457, -0.9712055325508118, 0.4270852208137512, 0.22426185011863708, 0.831221342086792, -0.8870702981948853, -2.3134922981262207, -0.27020373940467834, -0.6488902568817139, -0.9612461924552917, 2.6707146167755127, -0.05434567853808403, -0.7657480239868164, -0.6781557202339172, -1.6716364622116089, 0.2749159038066864, -0.7247098684310913], [1.2423346042633057, -1.2162114381790161, -0.5066167116165161, -2.464714288711548, 0.26358848810195923, -0.37629279494285583, -1.4421685934066772, 0.921477735042572, 0.09334421157836914, -0.27813562750816345, 0.6093566417694092, 0.25597405433654785, -1.1347620487213135, 0.08171357214450836, -1.6303685903549194, -1.8836194276809692, -0.5594058036804199, -1.6281795501708984, -0.23441952466964722, -0.316561758518219, 0.14205671846866608, 0.47753068804740906, -0.7235743403434753, 1.4421846866607666, 2.332859516143799, 2.0110292434692383, 0.7827330231666565, 0.02677486650645733, -1.0192714929580688, -0.07165390998125076, 1.9965486526489258, -0.3086024820804596, -1.66944420337677, -1.8582026958465576, -1.8160985708236694, -0.9187415242195129, -0.5842670202255249, 1.7428979873657227, -0.0742974504828453, -1.2050846815109253, 0.2943462133407593, 1.9398624897003174, -0.8294176459312439, -0.1597256064414978, 0.22721998393535614, 1.9069626331329346, 1.3266505002975464, 0.9086805582046509, 1.7713212966918945, 0.865088164806366, 0.7455442547798157, 1.4102580547332764, -2.177131414413452, -0.7302797436714172, 0.07073379307985306, -0.8011468648910522, -0.4778078496456146, -0.48804011940956116, 2.2528867721557617, -0.11808308213949203, -0.5701113343238831, -1.027744174003601, 0.6918261051177979, -0.6323533654212952, -0.08256645500659943, 1.345570683479309, 0.09425579011440277, 0.7311151027679443, -1.6896281242370605, 0.8267673254013062, -0.49218714237213135, -0.1965983361005783, -0.9964653253555298, 1.2324182987213135, -0.32579100131988525, 1.002626895904541, -0.21349121630191803, -1.0941303968429565, 1.0979583263397217, -1.2200998067855835, 0.42501378059387207, 0.16615413129329681, -0.5841394066810608, 1.7199122905731201, 1.1082721948623657, -1.3659480810165405, -0.29406020045280457, -2.1965088844299316, 0.0383475199341774, -2.3257129192352295, 1.0469281673431396, 0.7162013053894043, -1.471768856048584, -0.47130003571510315, 1.0149524211883545, -0.42011839151382446, -0.4546459913253784, -1.2286995649337769, 1.3304736614227295, -1.2942044734954834, -0.6487411856651306, -0.04542265832424164, 0.12525729835033417, -0.18465964496135712, -0.8062379956245422, -1.1008679866790771, 0.8979679346084595, -0.48982706665992737, -1.425523042678833, -1.1439762115478516, -0.8468969464302063, 0.6386964917182922, 2.1830904483795166, -0.6758313775062561, -0.8249437212944031, -0.3680219054222107, 0.20905378460884094, 0.8978037238121033, -0.5311604738235474, -0.7511839270591736, 0.4149515628814697, -1.5329034328460693, -0.2753383219242096, 2.1282167434692383, -1.2116986513137817, 0.26076918840408325, 0.1857263147830963, -0.5645031332969666]], [[-1.4283474683761597, 1.0831220149993896, 0.007567298598587513, -0.6704248785972595, -0.45061302185058594, -1.8638887405395508, -1.1088124513626099, 0.818952739238739, -0.22841356694698334, -1.4000834226608276, 0.650326132774353, 0.09046774357557297, -0.5417652726173401, 0.9309425354003906, 1.1862415075302124, 0.8365603685379028, 0.19467668235301971, 0.29299601912498474, -0.6684669852256775, -0.3866361677646637, 3.9376392364501953, -0.28109586238861084, -1.0577236413955688, -0.42143142223358154, -1.4530251026153564, 0.5717251300811768, -0.3023111820220947, -1.459928274154663, -0.412438303232193, 0.09686282277107239, -1.496863603591919, -0.6280257701873779, -1.5875858068466187, 2.006460666656494, 0.5909343957901001, -2.7872068881988525, -0.6999333500862122, 0.28397640585899353, -0.06219670549035072, -0.11493373662233353, 0.46214520931243896, -0.7020782828330994, -1.0315735340118408, -0.09063290059566498, 1.5719672441482544, 0.7017049193382263, 1.8810455799102783, 0.9839652180671692, 0.06643368303775787, 0.391260027885437, -2.060737371444702, -0.6897854804992676, -0.3281344175338745, -1.3524185419082642, 0.13714629411697388, -0.7111753225326538, -1.281527042388916, 1.2403011322021484, 0.3804946541786194, 0.47613099217414856, -1.0136741399765015, 2.159869432449341, 0.6848452091217041, 1.0287717580795288, -1.6386778354644775, 0.4772525727748871, 0.35792577266693115, -0.663530170917511, 0.30464306473731995, 0.607861340045929, -0.1923728734254837, -1.2008583545684814, 0.35934922099113464, -0.06460662186145782, 1.3896541595458984, 0.24320301413536072, -1.508735179901123, -2.2191708087921143, -0.6658815741539001, 0.3858279585838318, -1.011500358581543, 0.32462912797927856, -1.2211662530899048, -0.0762869343161583, 0.7503848075866699, -0.97520512342453, -0.2412053495645523, 0.6553575992584229, 0.8873063921928406, 1.0452970266342163, -1.0294475555419922, 1.7293404340744019, 1.5861918926239014, 0.5511841773986816, -0.30352911353111267, 1.1105698347091675, -1.0512886047363281, -0.9961901307106018, 0.3620765507221222, 0.07745544612407684, 0.5593163371086121, -0.08928569406270981, -1.0065752267837524, 1.0504170656204224, 1.4305403232574463, -1.8400170803070068, -0.5686086416244507, -0.16191831231117249, 0.7154256701469421, 0.670977771282196, -1.0482186079025269, 0.40476682782173157, 0.5980741381645203, -0.4995976686477661, -0.6586235165596008, -1.8638687133789062, -0.1656220555305481, -0.7204922437667847, -1.4414968490600586, 2.2991347312927246, 1.5020029544830322, 0.849474310874939, -0.6353751420974731, 0.006063912995159626, 0.08305349200963974, -0.7370128035545349, 0.037420812994241714, 1.4028135538101196], [-0.2748550474643707, 0.7261525988578796, 0.6987060904502869, 0.1651613414287567, 0.5718550682067871, 0.002339873695746064, -0.4984174370765686, 1.383484125137329, -1.4991246461868286, -2.1470911502838135, 1.843587875366211, -1.097101092338562, 2.1975655555725098, -0.17864862084388733, -1.791774034500122, -0.5138540267944336, 1.291983723640442, -1.698622465133667, -0.2652478516101837, -0.5112621188163757, 1.1091468334197998, -1.343095064163208, -1.5070693492889404, -1.0835626125335693, 0.500274658203125, -0.54137122631073, 1.1317733526229858, -0.2263329029083252, -1.204906702041626, 1.4928311109542847, 0.041005268692970276, -1.6893266439437866, -1.3039087057113647, -0.5225024819374084, -1.3079215288162231, -0.2956464886665344, -0.01913599483668804, 1.1847196817398071, -0.01791442558169365, 1.2105624675750732, 0.6405872106552124, -0.9179522395133972, 0.4995530843734741, -1.422498106956482, -1.0126370191574097, 2.277770757675171, 0.8171060681343079, -0.9263157844543457, -0.4919268786907196, -0.9492835402488708, -0.20918311178684235, 0.5302534103393555, 0.06363637000322342, -1.0263539552688599, -0.8330494165420532, 0.3610963523387909, 0.8495386838912964, -0.04751469939947128, -0.15752427279949188, -0.21253089606761932, -0.45046666264533997, -1.259134292602539, 0.14502264559268951, 0.20067238807678223, -0.2732752561569214, -0.9276608228683472, -1.3987467288970947, -2.5388433933258057, 1.0720772743225098, -2.4732589721679688, 0.10755913704633713, -0.6527605652809143, 1.1033812761306763, 0.7901930212974548, -1.87393319606781, 0.9163768887519836, 0.8657524585723877, 0.08117658644914627, -0.7064812779426575, -1.1340999603271484, 0.7545782923698425, -2.210150957107544, -1.1020840406417847, -0.9469968676567078, 0.6804123520851135, 0.4153953790664673, 0.02179880626499653, -0.6630627512931824, 0.45819351077079773, 1.3817179203033447, -3.253398895263672, -0.8993338346481323, 0.2747448682785034, 0.07010724395513535, 1.4170211553573608, 0.20635053515434265, 1.675998330116272, -2.601266860961914, -1.1098434925079346, -0.21365396678447723, 0.8584872484207153, -0.05666960030794144, -2.9765517711639404, 0.15284232795238495, -0.37821269035339355, 1.9929676055908203, 2.3579721450805664, -0.24770452082157135, -1.8581374883651733, -0.18170903623104095, -0.2964499294757843, -0.6118528246879578, -0.20404136180877686, -0.10888964682817459, -1.24214506149292, 0.07539896667003632, -0.28265833854675293, 0.0005165757029317319, -0.5463321805000305, -0.1841018795967102, -0.2858274281024933, -1.019502878189087, 0.5784997344017029, -0.420791357755661, -0.629482626914978, -1.2278623580932617, 0.9547656774520874, -0.022596167400479317], [-0.2812807261943817, -0.460716187953949, -0.7133038640022278, -0.4034349024295807, 0.5760881900787354, 0.07773400843143463, 1.354403018951416, -1.8523041009902954, 0.9667103886604309, -2.0070688724517822, 1.8358296155929565, 1.1150202751159668, -0.4929594397544861, -0.3094440698623657, -1.243959903717041, 0.5416337251663208, 0.3878230154514313, 1.1774705648422241, 1.135315179824829, -1.4730325937271118, -0.3339216113090515, -0.9597627520561218, -0.4121156632900238, -1.6783372163772583, 0.17717444896697998, -1.7520636320114136, -0.1800471693277359, 0.5873620510101318, -1.4612951278686523, 0.7142342329025269, -0.25542154908180237, -1.165810465812683, 1.7771227359771729, 0.24377764761447906, -0.36191898584365845, -0.6858904361724854, -0.6156489253044128, -0.9652677774429321, -0.48163220286369324, -0.21936677396297455, 0.7157776355743408, -1.0904862880706787, -0.4336549937725067, 0.7438775897026062, 0.15626740455627441, -0.06660757958889008, 0.8278766870498657, 0.8033081293106079, -0.1603546142578125, 0.5403344631195068, -0.9970043897628784, 1.8292968273162842, 0.5685335397720337, 0.6127650141716003, -0.2367691844701767, -1.277008056640625, 0.27846184372901917, 0.47368791699409485, -0.6273674368858337, -0.45486196875572205, -0.12658116221427917, 0.13110415637493134, -0.9653633832931519, -0.45652633905410767, -0.3178056478500366, 0.3526152968406677, 0.18652144074440002, -0.8942708969116211, -1.1378278732299805, 0.4231366217136383, -0.1476595401763916, -0.5266548991203308, -0.49681392312049866, 1.1029961109161377, 0.8116625547409058, 1.5593012571334839, -1.5171151161193848, 2.525585412979126, -0.8228410482406616, -0.6346836090087891, 1.505733847618103, -1.1869359016418457, 2.1190335750579834, 1.4074976444244385, 0.01255129650235176, -0.7418448328971863, 0.3186756372451782, -2.108311176300049, -1.4757951498031616, 1.2581411600112915, 0.10338499397039413, -0.8746140003204346, 2.187930107116699, -0.9228023886680603, -0.6235016584396362, 0.8660044074058533, -1.7108112573623657, 0.9055431485176086, 0.7160384654998779, 0.7744117975234985, -0.012736734934151173, 0.0234576053917408, -0.19739244878292084, 0.2291073501110077, 0.3603849411010742, -0.6246797442436218, 1.6475684642791748, -0.08409297466278076, 0.8485368490219116, 0.5754507184028625, 0.7452030181884766, -0.3487889766693115, -0.26942628622055054, -0.3712672293186188, 2.279146671295166, -0.7178519368171692, 1.5493123531341553, -0.9819775819778442, -1.0086171627044678, -0.13249537348747253, -0.3162126839160919, 0.6867967247962952, 0.1819727122783661, -1.950160264968872, 0.05497628450393677, 1.4928432703018188, -0.08352267742156982, 0.15567627549171448], [1.2804726362228394, 1.5670874118804932, -0.28361839056015015, -0.38798248767852783, -0.4379860758781433, -0.9461168646812439, 0.2892191410064697, -1.2721397876739502, 0.6962611079216003, 0.4869334399700165, 0.02679450996220112, 0.8388121128082275, 0.5284463763237, 1.7416284084320068, 1.290755033493042, 0.7848308086395264, -1.691786766052246, -0.5163753628730774, -1.7754783630371094, -1.3049581050872803, -1.004552960395813, 1.793541431427002, -0.2742253243923187, 0.37896546721458435, -0.48906734585762024, -0.4508046805858612, -0.6078041195869446, -0.32603195309638977, -0.4147247076034546, -0.616309404373169, -0.40159881114959717, -0.22994178533554077, 1.6579002141952515, -0.3113689720630646, -0.3350038528442383, 1.014892816543579, 0.7899511456489563, 0.18628518283367157, 0.5292956829071045, -0.6352908611297607, 2.725268602371216, -0.11939406394958496, 1.052656650543213, 1.6514556407928467, 0.9432389736175537, 0.8661684393882751, -1.0216785669326782, 0.8283771276473999, 0.6901783347129822, -0.5698168277740479, 0.4035249352455139, 0.6754706501960754, 0.252318799495697, -0.6226088404655457, 0.5269504189491272, -0.2956799566745758, -1.6282684803009033, 0.2484983503818512, 0.6236444711685181, 2.1531312465667725, 0.5396431684494019, 0.5622208118438721, 0.5922186374664307, -1.3217058181762695, 0.04723619669675827, 0.33598652482032776, -1.6493465900421143, -0.36562711000442505, 3.2815566062927246, 0.6375070214271545, -0.2952171862125397, 1.5211752653121948, 9.197987674269825e-05, -1.0117682218551636, -2.0300278663635254, 0.022219279780983925, -0.34676843881607056, -0.8945482969284058, 0.29196956753730774, -0.8938431143760681, 0.6785231828689575, 0.10387736558914185, -0.23485897481441498, -0.746505081653595, -0.532161295413971, -0.24283234775066376, -0.3641360104084015, 1.526107668876648, -1.1419541835784912, -1.0329244136810303, 1.3809764385223389, -2.6008036136627197, -1.0199522972106934, -0.36543071269989014, -0.32656314969062805, -2.002786636352539, -0.8769312500953674, 1.2592710256576538, 0.07298596948385239, -0.9688050150871277, -0.5512431263923645, 0.3852553069591522, 0.3222583830356598, 1.2200291156768799, 0.5640065670013428, 0.46332263946533203, 0.9686527848243713, -0.4240803122520447, 1.249031901359558, -1.0366175174713135, -1.0150011777877808, -0.3829331696033478, 0.36055731773376465, -0.6055535078048706, -1.0458204746246338, -0.1896822452545166, 0.3875049650669098, -0.1837398260831833, 0.8993557691574097, -0.5971378087997437, -0.3609413206577301, 0.41663357615470886, 0.4216513931751251, 0.483368456363678, 0.2703039050102234, -0.7918746471405029, 0.9226275682449341, -0.030927741900086403], [0.9919455647468567, 1.842614769935608, 0.09860209375619888, 0.6357336640357971, -1.0803622007369995, 1.2676178216934204, -0.9348496794700623, -1.644757628440857, -0.5210252404212952, 1.6182376146316528, 0.7428778409957886, 0.682266891002655, -1.031577467918396, -0.9715456366539001, -2.215473175048828, -1.8215211629867554, 0.6243005990982056, -1.6940703392028809, -0.3600347936153412, 0.7681946158409119, -0.5490999221801758, -0.8987985849380493, 0.04741397500038147, -0.2528076171875, -0.9264397025108337, 0.8237566351890564, 0.6278351545333862, 0.344419002532959, -1.4826200008392334, -0.6078395247459412, -0.10295356065034866, 1.345257043838501, 0.19206705689430237, -1.8738360404968262, 0.3716910183429718, 0.4453789293766022, -0.27558282017707825, 1.3203307390213013, -0.04103008285164833, 0.9231510162353516, -0.08063352108001709, -2.501079559326172, -0.692467212677002, 0.06208484247326851, 1.310003399848938, -0.4555794894695282, 1.1985948085784912, 0.7128579020500183, 0.0948539674282074, 0.08260639011859894, 1.0866882801055908, 0.1279640644788742, 0.24116148054599762, 1.0373879671096802, -0.3506854176521301, -0.39910149574279785, -1.359879970550537, 0.9208108186721802, -1.2155299186706543, -0.8481910228729248, 0.004676932003349066, -1.0852018594741821, 1.4981848001480103, -1.1017804145812988, -1.0454463958740234, -0.6363236904144287, -1.6434766054153442, 0.04500999301671982, 0.19185441732406616, -1.0987560749053955, -1.2827454805374146, -0.7239159941673279, -1.1839237213134766, -0.1671990305185318, 2.695291042327881, -0.01389367040246725, 0.16307885944843292, -0.08316415548324585, 2.3842129707336426, 1.241995930671692, -0.8142882585525513, -0.4346499443054199, 0.9172391891479492, 1.4408149719238281, -1.6460891962051392, -1.6616847515106201, -0.853570818901062, 0.8953693509101868, -0.1367194801568985, -1.7478524446487427, -0.5012921094894409, -0.16107137501239777, 0.8282551765441895, -0.10869855433702469, -0.8233973383903503, 0.5224446058273315, -0.16901031136512756, -1.3396576642990112, -2.7521679401397705, -1.1729962825775146, -0.6281500458717346, -1.9275813102722168, 0.7312303185462952, -0.526043713092804, -0.059357114136219025, 0.9082556962966919, -1.1960430145263672, -0.0834767073392868, 0.6483311653137207, 0.5177589058876038, 1.2936466932296753, -1.519906997680664, 0.9412474036216736, -0.4053313434123993, -0.7047193646430969, 0.2276211380958557, -0.23310579359531403, -1.4642683267593384, 0.7351301312446594, -0.5463987588882446, -1.694314956665039, -0.7929778099060059, 0.19607307016849518, -0.2689991891384125, 1.1947630643844604, 0.17766238749027252, 1.3468278646469116, -1.544793725013733], [0.550861120223999, -1.6002837419509888, 0.8509880900382996, -1.5018185377120972, 1.1026782989501953, 0.7706848978996277, -0.23116283118724823, 0.28843697905540466, -1.114365816116333, -0.9726561307907104, -1.1265777349472046, -2.609712839126587, -0.6308327913284302, 0.39090731739997864, -0.6608162522315979, 0.4858781695365906, 0.13199856877326965, -0.7492061853408813, 1.1325278282165527, 1.1204599142074585, 1.4491634368896484, -1.3689510822296143, -2.078169584274292, 0.9446549415588379, 2.4169628620147705, 0.9678401350975037, 0.6693034172058105, 2.1857762336730957, 0.8922471404075623, 0.7762678861618042, 1.1544020175933838, 0.2662062644958496, -1.69580078125, -0.9875727891921997, 1.7492694854736328, -0.32912448048591614, -0.11072275042533875, -0.6520245671272278, -0.43253082036972046, 0.4281599223613739, -1.1095664501190186, 1.1899797916412354, -0.2997084856033325, -1.9625396728515625, 0.8857249617576599, 0.061536502093076706, -0.49126356840133667, -1.626128911972046, 0.6406484246253967, -1.0793458223342896, 1.671639084815979, -1.7053298950195312, -0.723975658416748, -1.5188778638839722, 1.7840615510940552, -0.29203158617019653, -0.5552130341529846, 0.5077995657920837, 0.2942526042461395, -0.7564875483512878, -1.664258599281311, 1.0242376327514648, -0.21377350389957428, -0.5275185108184814, 0.4759398102760315, 0.20695088803768158, -0.8001552820205688, -0.7672715783119202, -1.3540325164794922, 0.45792558789253235, -1.5143187046051025, -1.7767304182052612, -0.7056446671485901, 1.1215718984603882, 0.5466128587722778, -0.09006469696760178, -1.885412335395813, 0.6303218603134155, 0.7030019760131836, 0.15696583688259125, 0.18571199476718903, -1.269208550453186, 0.38059231638908386, -0.9301297664642334, -0.9188371896743774, 2.382810592651367, 1.0432997941970825, -0.3148922920227051, -0.9238382577896118, 0.6543007493019104, -0.5658558011054993, 1.948091745376587, 0.6496879458427429, -0.35249483585357666, -0.6878600120544434, 1.4369536638259888, -0.23359860479831696, 0.03489214926958084, 0.5591263771057129, 1.3019640445709229, -0.055531568825244904, -0.8215305209159851, -0.41141650080680847, 0.032810937613248825, -0.007857056334614754, -0.7786767482757568, -2.415652275085449, 1.187583088874817, 1.1341553926467896, 0.22026623785495758, -1.6418819427490234, 0.7502533793449402, 0.6436347365379333, 0.09139896184206009, 0.45286011695861816, -0.5800750255584717, 1.5381649732589722, -0.0029510040767490864, -0.6134482622146606, -0.276305228471756, -0.3066360056400299, 0.2286471575498581, -0.030357759445905685, 0.8872854113578796, -0.6917077302932739, -0.5274263620376587, -0.1609947383403778, 0.1807178109884262], [0.7248056530952454, -0.5456179976463318, -0.22345399856567383, 1.3912166357040405, 0.8419362306594849, -0.9963493347167969, 0.43896934390068054, 2.3026435375213623, 0.5490953326225281, 1.1601625680923462, -1.0078235864639282, 0.5919238924980164, -0.16736888885498047, 0.25623300671577454, -0.10893090069293976, -0.7785311937332153, 0.5755234360694885, -1.3319131135940552, 0.10245367139577866, 0.7410807609558105, -0.7143188714981079, -0.389880508184433, -0.6998820900917053, -1.129866123199463, 1.7011457681655884, 0.6330628395080566, 0.5350289940834045, 0.5394552946090698, -1.2620971202850342, -0.7728578448295593, -1.719545841217041, 0.15497443079948425, 0.1398739069700241, -0.04884321615099907, 1.5088845491409302, 0.4104461073875427, 1.1480299234390259, 2.4608469009399414, -0.4876594543457031, -1.6340867280960083, -0.03094911016523838, 0.6574798226356506, -0.3153113126754761, -0.4346050024032593, -0.985179603099823, -1.4298995733261108, -1.3590366840362549, -0.21032387018203735, -0.6457189917564392, 0.9786885380744934, -1.6548210382461548, 1.043691635131836, -0.3256223797798157, -1.1443549394607544, -1.116874098777771, -1.3305728435516357, -2.3743722438812256, -0.06639131158590317, -0.2455197423696518, 1.3084677457809448, -1.6012500524520874, -0.9339758157730103, -2.8097877502441406, 0.10385604202747345, 1.1640739440917969, -0.3284395635128021, -0.21627676486968994, 0.15208002924919128, 0.10233495384454727, -0.48516494035720825, -0.8381925821304321, -0.8333516120910645, -0.6044234037399292, -1.0616483688354492, -0.6041028499603271, -0.9794935584068298, 1.1345610618591309, 1.5001113414764404, 0.5647119879722595, -0.2863600254058838, -0.45706743001937866, -0.8642526268959045, -0.9769030213356018, 0.3153812289237976, -0.23159906268119812, 0.7648696303367615, 0.4859463572502136, -1.985399603843689, 1.184162974357605, 1.576770544052124, -2.347191095352173, -0.864935040473938, -0.41968652606010437, 0.4751378297805786, -0.34460142254829407, 0.5118546485900879, 1.071670413017273, -0.14005498588085175, 0.6771440505981445, 0.36734652519226074, -0.7376047372817993, 0.316240519285202, 0.28860530257225037, -1.7131764888763428, 0.613123893737793, -0.13591048121452332, 0.17501342296600342, -0.5812423229217529, -0.40856054425239563, -1.024337649345398, 1.2467725276947021, 0.14823845028877258, 0.08765818178653717, -1.6284034252166748, 0.03458718582987785, -0.11585395783185959, -1.2175847291946411, 0.46361902356147766, -1.1325637102127075, -0.5548372268676758, 1.3747813701629639, 0.6213403344154358, 0.1669289618730545, -1.61506986618042, -2.1281590461730957, 0.13810408115386963, -0.5168895721435547, 0.7754406332969666], [-0.3369387090206146, 1.0584132671356201, -1.7154594659805298, 0.5022520422935486, 1.2828152179718018, 0.8083357810974121, -0.6187667846679688, 1.1217414140701294, 0.07237661629915237, -0.7009553909301758, 0.366439551115036, 1.6417932510375977, -1.3400369882583618, 1.097309947013855, -0.12957866489887238, -1.084322452545166, -0.2232362926006317, -1.6625468730926514, 0.2326069325208664, 0.14168551564216614, -0.08169714361429214, -0.43660929799079895, 0.6627531051635742, -0.16719244420528412, 1.3453999757766724, -0.41082966327667236, 0.8699740171432495, 0.5348884463310242, -0.1597922146320343, -1.1210756301879883, -0.5872906446456909, 0.37765172123908997, -0.18247582018375397, -0.016657283529639244, -0.8324187397956848, 1.5891560316085815, 0.880560040473938, -0.7337680459022522, 0.81191086769104, -0.18782982230186462, 0.40129512548446655, 0.6791019439697266, -1.5538458824157715, -0.9715026617050171, -0.8712924122810364, -1.4547926187515259, 0.7493993043899536, -0.565666675567627, 0.023060940206050873, -0.00292418384924531, -0.18639487028121948, 0.5292009115219116, -0.009078814648091793, 0.13229331374168396, 1.4415912628173828, -0.2853788137435913, 0.5278316140174866, 0.3072870075702667, -0.08130543678998947, -1.244955062866211, -0.8608112931251526, 0.22398899495601654, -0.5313779711723328, 0.7688704133033752, -0.5876219272613525, -1.3205757141113281, 0.5268166065216064, -0.800696849822998, -0.5170577764511108, 0.08013484627008438, 0.842288076877594, -0.4913158714771271, 0.3616512715816498, 0.7362803220748901, -1.8915084600448608, 0.37826284766197205, -0.4993915259838104, 1.561466932296753, -0.8982223272323608, -0.8754684925079346, 0.9864291548728943, -0.8135777711868286, 0.7574375867843628, -0.6604925990104675, 2.735363483428955, 0.5459116697311401, 0.41071754693984985, -1.211606740951538, 0.3221908211708069, 0.6547085046768188, -0.8582555651664734, 0.15840761363506317, 0.284630686044693, 0.25945210456848145, -0.3835330605506897, -2.31074857711792, -0.11762133240699768, 0.8707194328308105, -0.26560473442077637, -0.729015052318573, 0.5353963971138, -0.1611979901790619, -0.6353667378425598, -0.14139388501644135, 0.699439287185669, -1.5191911458969116, 0.5657088160514832, 0.1777634471654892, 1.431455373764038, -1.9067833423614502, 0.17750702798366547, -1.5859116315841675, -0.19095216691493988, 0.8300084471702576, 1.1598130464553833, 1.2196838855743408, -2.0052542686462402, 1.3357654809951782, -0.5050818920135498, 0.092755027115345, 0.9437501430511475, 0.22666189074516296, -0.14206768572330475, 1.5694401264190674, 1.416397213935852, 0.5637969970703125, -0.07400213927030563, -0.4619148373603821], [-0.041365232318639755, -0.34301450848579407, 0.05978773534297943, -0.2632668614387512, 0.3436969816684723, -0.6993598341941833, -1.2570397853851318, 1.2498793601989746, 0.49189889430999756, -0.04277767986059189, -0.4582500755786896, 0.6099244952201843, -1.9233217239379883, 0.5544061064720154, -1.3655104637145996, 0.2316139042377472, -0.8585172295570374, -1.2703770399093628, 0.8868455290794373, 0.39926570653915405, -2.1977343559265137, -1.7887576818466187, -0.6845226883888245, -2.0002896785736084, -0.11052866280078888, 0.2889872193336487, 0.33164894580841064, -0.041798584163188934, 1.467753291130066, -0.5601311326026917, 0.03366474062204361, 0.6874080300331116, -0.6756943464279175, 0.8093962669372559, -1.8014742136001587, -1.3053898811340332, -0.021749136969447136, -1.113850474357605, -0.7713532447814941, 0.31963813304901123, 0.7120182514190674, -0.39192163944244385, 0.49960631132125854, 0.09063529968261719, -0.23077307641506195, 1.2942990064620972, -0.5514940023422241, -0.19332599639892578, 0.491067111492157, -1.6401420831680298, 0.8277313113212585, 1.9647384881973267, -0.5448150634765625, 0.19611793756484985, 0.38264209032058716, 1.628449559211731, -2.0200297832489014, -0.053761668503284454, -0.20515522360801697, -0.47152212262153625, 0.14981135725975037, 0.0806911438703537, -0.07003474980592728, 0.2845921516418457, 0.7243254780769348, 0.589436411857605, 0.15964804589748383, 0.3278924226760864, -1.5448188781738281, 1.6360169649124146, -1.0054571628570557, -0.522983968257904, 0.22839397192001343, -0.4944295287132263, -0.3531050682067871, -0.35051873326301575, -0.7068252563476562, -1.9567080736160278, 0.3571159243583679, 1.334775447845459, -1.3190149068832397, -0.8284181952476501, -1.228769302368164, -0.23340100049972534, 1.1950467824935913, 0.6913885474205017, 0.9349039793014526, -0.3544975817203522, 0.1143743023276329, 0.2353680431842804, 0.4321447014808655, -1.0688319206237793, -2.526703119277954, -1.4897446632385254, -0.15614621341228485, -0.6385815143585205, -0.28328242897987366, 0.2424265742301941, 0.27362242341041565, -0.2757747769355774, 1.321567177772522, -1.402274250984192, 0.3808990716934204, -0.049970731139183044, -1.546664834022522, 2.3200559616088867, 0.18246695399284363, 0.17459554970264435, -0.9571930766105652, 0.1953653246164322, -0.533001720905304, -0.7675344347953796, 0.601388156414032, 0.5823842287063599, 1.4690111875534058, -0.21567854285240173, -0.03519187122583389, -2.954686403274536, -0.23701463639736176, 0.5440512299537659, -0.4409295320510864, -0.7267302870750427, -1.3266572952270508, 0.762231171131134, 0.1551067978143692, -2.369915246963501, 2.318450927734375, 0.4435485005378723], [-2.5867979526519775, 2.165710210800171, -1.2611701488494873, -0.5774458050727844, -2.4617629051208496, -0.3285483717918396, 0.3378662168979645, 1.0729577541351318, 0.6885895729064941, -0.12746813893318176, -0.05257254093885422, -0.27483364939689636, 0.40964600443840027, -0.6736696362495422, -0.6665399074554443, -1.20616614818573, 0.8981523513793945, 1.120139718055725, 0.5294146537780762, 0.000978716998361051, -0.12071641534566879, 1.0522263050079346, -1.1704046726226807, -0.07418916374444962, -1.1962207555770874, 0.3763044774532318, 1.159757375717163, -0.5529072880744934, 1.8196285963058472, 0.11182938516139984, -0.23480810225009918, -0.6624782681465149, -1.8080042600631714, -0.4828922152519226, 1.1548882722854614, 0.3822751045227051, -0.06876382976770401, -0.26765480637550354, 0.329710453748703, -2.0977895259857178, -0.3074774742126465, -0.9984142780303955, -0.8284130692481995, -2.2910540103912354, 0.8522553443908691, -0.060591138899326324, 0.31773313879966736, -0.7381564974784851, -0.05000689998269081, 0.4995623528957367, -1.5634323358535767, 2.1305649280548096, 0.1254928559064865, 1.6390392780303955, 0.6724602580070496, 2.101118564605713, 0.34634602069854736, 0.4575996398925781, 0.7893637418746948, 0.09576676785945892, -0.6492459774017334, -1.2864632606506348, 1.3481334447860718, -0.9081732630729675, 0.5240689516067505, 0.3150476813316345, -1.478954553604126, 1.8383245468139648, -1.0079559087753296, 1.2550387382507324, 0.32550862431526184, 0.34564703702926636, -0.31992337107658386, -0.22225068509578705, -1.311745047569275, 0.18088671565055847, -1.4727692604064941, 0.8224987983703613, -1.123644471168518, -0.7238965630531311, -0.28768429160118103, 0.08532809466123581, 0.4474952816963196, -0.18262910842895508, -1.3802894353866577, 0.5426697731018066, -0.3309212625026703, 0.45847901701927185, 0.11625099927186966, -0.23381859064102173, -0.7874758243560791, 0.5832914113998413, 0.16951143741607666, 0.5251147747039795, -0.3723224401473999, 0.3968941271305084, -1.1385514736175537, -1.1537059545516968, 0.4358040392398834, -0.9326266646385193, 0.35822755098342896, -1.1009544134140015, -0.763785719871521, 0.44343164563179016, -0.40342164039611816, -0.6049925684928894, 0.17378468811511993, -0.632345974445343, -1.034940481185913, 0.3283335566520691, 0.3544081449508667, -0.5932481288909912, 2.4199230670928955, -1.0172580480575562, 0.22941362857818604, 0.9241234660148621, -1.144943356513977, 0.2586393356323242, 0.2021239846944809, 0.8585061430931091, 0.05562112480401993, -1.338793396949768, -0.2420685738325119, 0.5217000842094421, -0.5568668246269226, 0.9157915711402893, 0.5481014251708984, -0.3236669600009918], [-0.10354503244161606, 0.29412946105003357, -0.18434253334999084, -0.12347371876239777, 0.07668006420135498, 0.4514695107936859, -0.2740101218223572, -0.8587133884429932, -0.16937820613384247, 0.7193369269371033, -1.0725939273834229, -0.21344570815563202, 0.008405872620642185, -0.008873985148966312, 0.522941529750824, -0.22559812664985657, -0.42232486605644226, 1.5030781030654907, 1.3066271543502808, 0.34890130162239075, -0.135647714138031, 2.179981231689453, -1.5080125331878662, -0.08538171648979187, -0.15543609857559204, -0.6885639429092407, 0.9751752018928528, 0.6099752187728882, 0.9930353760719299, 1.2157288789749146, -0.33666881918907166, 1.1469756364822388, -0.6782821416854858, -0.6534510254859924, -0.9906182289123535, 2.6741230487823486, -0.9211770296096802, 0.8917542695999146, -0.8442416191101074, -0.5823677778244019, -0.1462264508008957, -0.8798708915710449, 0.22258363664150238, 0.5978925824165344, 0.43940040469169617, 1.65609610080719, -0.6169466972351074, -1.24032461643219, 2.1005654335021973, -0.360302597284317, -1.005704402923584, -0.42439907789230347, -0.5293657779693604, 0.33513420820236206, 1.2747310400009155, 0.5318343639373779, 1.3376412391662598, 0.9903036952018738, 0.4243336617946625, -0.964992344379425, -1.2894505262374878, 1.1577237844467163, 0.8448042869567871, 0.9683451056480408, -0.7047789692878723, 0.55843585729599, 0.5041615962982178, 0.8341549038887024, 0.9921752214431763, 0.4104660749435425, 0.399111807346344, 1.5223522186279297, -0.4609954357147217, -0.6622853875160217, 0.8873960971832275, 0.4645293653011322, -0.5628014206886292, -2.1618502140045166, 0.8889533877372742, 0.05323706194758415, -0.5069707036018372, 1.2576910257339478, 0.21073170006275177, 1.257334589958191, 1.311318278312683, 0.367707222700119, 0.3515179753303528, 0.9116812944412231, -0.4953472316265106, -0.03200880065560341, 0.9111045002937317, -0.5480480194091797, -1.862838864326477, -0.6453122496604919, 0.6871121525764465, -0.5104718208312988, -0.01574043743312359, 0.18757525086402893, 0.5583573579788208, 0.238168403506279, 2.3829338550567627, -0.42932289838790894, -0.04452865570783615, 0.7241389155387878, -0.9709708094596863, 1.1322102546691895, 1.9046066999435425, 0.3788287341594696, 2.822603940963745, -0.47516921162605286, 0.4597824215888977, -0.037461623549461365, 0.9829756617546082, -0.6192866563796997, -0.23161593079566956, -0.8936046361923218, 0.5358874201774597, 0.41874414682388306, -0.6483897566795349, 1.8005635738372803, -0.4689296782016754, -0.10383392125368118, -2.084151029586792, 0.9332906603813171, -0.30967435240745544, 0.8204983472824097, -0.777264416217804, 1.0275475978851318], [-0.9629566073417664, -1.2487508058547974, 1.0756759643554688, 0.20486462116241455, -0.6911641359329224, -0.12983635067939758, 1.35383939743042, -0.28377410769462585, 2.340914726257324, 0.026747960597276688, 0.9357924461364746, -2.42299222946167, -0.07078344374895096, 0.0383017435669899, -2.6237165927886963, -0.33382898569107056, 0.7593532800674438, -0.4863249361515045, -1.6211135387420654, 1.5575753450393677, 0.9681898951530457, 0.05651938542723656, -1.0759074687957764, 0.8797066807746887, 1.1213120222091675, 0.26187992095947266, -0.05419964715838432, -0.6312747001647949, -0.8302932977676392, -1.9149733781814575, -0.35160914063453674, 0.12560096383094788, -0.11089321225881577, -1.2267658710479736, 0.5528278350830078, -0.2601185441017151, -1.4126065969467163, 0.5346100330352783, -0.6782037019729614, -0.2015266716480255, -0.15073591470718384, -1.7910747528076172, -0.2447267770767212, -1.2554142475128174, 0.9635255932807922, 0.6202710866928101, 0.5043664574623108, 0.16962043941020966, -1.0673561096191406, 1.2443934679031372, -0.8114075660705566, 0.13381843268871307, 1.35172438621521, -0.5466219782829285, 0.6925360560417175, 0.3143152892589569, 0.9583421349525452, -0.9638684988021851, -0.9189579486846924, -1.506608009338379, -0.7009797692298889, -0.51751708984375, -0.42766690254211426, -0.3721121847629547, 0.40600714087486267, 0.3968900442123413, -0.6340023875236511, -1.215836763381958, 0.6494617462158203, 1.2121855020523071, -0.5698261857032776, -0.6212264895439148, 0.9246649742126465, -1.1444371938705444, 0.9839890003204346, -1.8316706418991089, 1.8985098600387573, -0.4044269621372223, 0.4184136390686035, 0.9566542506217957, 0.2977762818336487, -0.2204405665397644, -0.014506793580949306, 0.4065355062484741, 0.10340700298547745, -0.281265527009964, -0.12026279419660568, -0.15509182214736938, 0.8099395632743835, -0.3427794873714447, -0.8188617825508118, 0.6014295220375061, 1.240767478942871, -0.7226454615592957, -0.5206502676010132, -0.774203896522522, -1.0653184652328491, -0.5348812341690063, -0.9074638485908508, 2.6687841415405273, 1.3595014810562134, 2.6338603496551514, -0.020444653928279877, 0.32229045033454895, -0.736108124256134, 0.13332097232341766, -1.0461846590042114, 0.5258182287216187, 1.7271206378936768, 0.6716241836547852, 0.49832719564437866, -0.4620799720287323, -0.2908303439617157, -0.2914009094238281, 0.28342241048812866, -0.7617587447166443, 1.443199872970581, 0.5098914504051208, 0.4028628468513489, -1.496402621269226, 1.1307495832443237, -0.4831899404525757, 1.2466524839401245, -0.2907028794288635, 2.0505149364471436, -0.5640110373497009, 0.21856854856014252, 0.7084842920303345], [0.3210437595844269, 0.2116064727306366, 0.8160446286201477, 0.5936971306800842, 0.045389145612716675, -0.132359579205513, -0.33325886726379395, -0.5441915392875671, 1.3342279195785522, -0.9352295994758606, -1.029632806777954, -0.02418399602174759, 0.3396925926208496, 0.7745457887649536, -0.17004218697547913, 0.6638040542602539, -0.12444635480642319, -0.8060743808746338, -0.024258315563201904, 2.0747454166412354, -1.4130345582962036, 0.539874255657196, 0.7272090911865234, -0.8240043520927429, 0.10970252752304077, -0.02711065299808979, 0.429400771856308, -0.4375651776790619, -0.18057163059711456, -0.3868454694747925, -0.02814040146768093, 0.8189793825149536, -1.3399542570114136, 0.281333863735199, 0.21791872382164001, 0.9361302256584167, -1.0449374914169312, -0.5748921632766724, -0.2172294706106186, 1.357716679573059, 0.32613515853881836, 1.4777915477752686, 0.034553200006484985, -1.5322567224502563, 1.5016562938690186, -0.755557119846344, 2.0094335079193115, 0.3465586006641388, 0.21720226109027863, -0.3810451924800873, 0.0466747023165226, -0.39501941204071045, 1.7659077644348145, 0.07609134912490845, 0.4093439280986786, 0.18269532918930054, -0.0001166680158348754, 0.19858595728874207, 1.9132630825042725, 1.5754692554473877, -0.4716523289680481, 1.4509447813034058, 0.31100907921791077, 0.908724844455719, 0.00902000442147255, 0.814871072769165, 0.35417890548706055, 0.25626063346862793, 0.17047318816184998, 0.4029124677181244, -1.3555879592895508, 0.865553081035614, 1.153219223022461, -0.2553037405014038, 1.688918948173523, -0.9589047431945801, 0.5877072811126709, 0.5960062146186829, 0.6900021433830261, -0.00842125341296196, -0.05201224237680435, -1.8448776006698608, -0.5504817962646484, -0.8686283826828003, 1.4624446630477905, -1.7912986278533936, 0.7565803527832031, 1.8516156673431396, -0.2397928088903427, 0.06477654725313187, 1.4879579544067383, 0.06324036419391632, 1.0491652488708496, 0.26561710238456726, 0.8544726967811584, 0.9677030444145203, -0.31439122557640076, 2.4419710636138916, -0.7292569279670715, -1.0239230394363403, -1.540238380432129, -0.9334604144096375, -0.6106712818145752, 1.3448785543441772, -0.8582161068916321, 0.05511295795440674, -0.03538913279771805, -0.6856025457382202, -1.621323585510254, -1.577170729637146, 1.4259650707244873, 0.7729493379592896, -1.0478973388671875, -0.34506741166114807, 1.1725404262542725, 1.0801056623458862, -0.7085813879966736, -1.364458441734314, 1.5703456401824951, -1.814133644104004, 0.7547795176506042, 1.4749826192855835, -1.4047961235046387, -0.9102084040641785, -1.673475742340088, -0.5997888445854187, -0.43505293130874634, -0.2913374602794647], [-0.9806604981422424, -0.7032353281974792, -0.8611248731613159, 1.142124891281128, -0.36264944076538086, -1.1649922132492065, -0.6742696166038513, -0.9369685649871826, 0.981367826461792, -0.500686764717102, -1.4199079275131226, 0.8502592444419861, -1.7228649854660034, -0.14159870147705078, -0.1952272653579712, 0.4218319356441498, 0.21460483968257904, -1.7503468990325928, -0.016873663291335106, 1.68923020362854, -1.3909236192703247, 0.5774984359741211, 1.30491304397583, -1.367021918296814, -0.0960884839296341, -1.3674873113632202, 0.21870848536491394, -1.4753445386886597, -0.19954663515090942, 2.4270873069763184, -0.11672750115394592, -0.11275875568389893, -0.15888693928718567, -0.11327949911355972, 1.1819560527801514, 1.638641357421875, -0.13652493059635162, 1.7634555101394653, 0.6531078219413757, -0.42519739270210266, 1.0972775220870972, -0.52412348985672, -0.5925520658493042, 0.45715129375457764, 1.183045506477356, -0.482876181602478, -0.6052656173706055, 0.35744377970695496, 0.35144054889678955, -1.0564552545547485, 1.2697184085845947, 0.7988793253898621, -0.21751952171325684, 1.1384576559066772, 0.35883456468582153, -1.2922757863998413, 1.128826379776001, 0.841160774230957, -0.5282521843910217, 0.3267502188682556, 1.2119742631912231, 1.2612364292144775, -0.299893319606781, 2.1736881732940674, 0.8842195272445679, 1.8255698680877686, -1.2733134031295776, -0.2806989848613739, 0.15311717987060547, 0.27713868021965027, 1.4471131563186646, 0.5793464779853821, -0.03397383168339729, 1.327575922012329, -0.0890255942940712, 0.9039316177368164, -0.26382166147232056, -0.4866141378879547, 1.4905295372009277, -0.9924526214599609, 0.5845121145248413, 0.9714452028274536, 0.5691196918487549, -0.32734885811805725, -1.0480096340179443, 0.07983803749084473, 0.5700657367706299, 0.5010905861854553, -0.35993948578834534, 0.17533034086227417, -1.6950483322143555, -0.0475224107503891, -1.1853841543197632, 0.257685124874115, 0.5542395114898682, -0.5032546520233154, 1.258408546447754, 0.1588549017906189, 0.4089103937149048, 0.10262523591518402, 0.2462090402841568, -0.8845040798187256, -0.20255056023597717, 0.5238880515098572, 0.3716188967227936, -0.1409902274608612, -0.5088094472885132, 2.1061525344848633, -1.1752599477767944, -0.43715107440948486, -0.21768724918365479, -0.4700923562049866, 1.206098198890686, -1.2507736682891846, -1.2328290939331055, -1.321923851966858, -1.1952874660491943, 0.5432929396629333, 1.277353048324585, 0.3784726858139038, -1.376186490058899, -1.7119619846343994, 0.33929163217544556, -0.9260379076004028, -0.46680372953414917, -1.9247996807098389, -0.5146417617797852, -0.5768449902534485], [-1.824592113494873, -1.0784357786178589, 1.4714020490646362, 1.1698108911514282, 0.5902985334396362, 1.4920202493667603, 0.4847268760204315, -1.090524673461914, -0.17817552387714386, -0.13206055760383606, 0.23423781991004944, 1.0227144956588745, -0.7324624061584473, 0.5521368384361267, -0.6669858694076538, 1.4218101501464844, 0.831485390663147, -0.7110017538070679, -0.4201086759567261, -1.4592912197113037, -0.4265805780887604, -0.16464056074619293, 0.14387205243110657, -0.297247976064682, -2.317526340484619, -0.6590131521224976, -0.505096435546875, -1.2007759809494019, -0.16552592813968658, 1.7940738201141357, -0.26517006754875183, 0.35469475388526917, -2.2325189113616943, 0.15682464838027954, 0.26048871874809265, 0.34862837195396423, -0.030436333268880844, 0.7405903935432434, 0.4541916251182556, 0.20292623341083527, -0.9934574961662292, 0.19182445108890533, 2.8778607845306396, -0.9849515557289124, -1.211347222328186, 1.0331875085830688, -0.5780856013298035, 0.6648446321487427, -0.11069007962942123, -0.3881417512893677, 0.27960434556007385, 0.2875257432460785, 0.40396639704704285, 0.5693721771240234, -0.9868724346160889, -1.583816409111023, 0.11044012010097504, 1.0675263404846191, 1.1376782655715942, -1.9806307554244995, 0.3410428464412689, 0.7093093991279602, 0.6015480756759644, 1.1290069818496704, 0.4331319034099579, -0.4372483193874359, -1.0893213748931885, 0.49821409583091736, 0.9310609102249146, -0.023894859477877617, -1.1124275922775269, 0.7427234649658203, 0.7634314298629761, -1.8848774433135986, -0.105802021920681, 1.8825569152832031, -1.1546555757522583, -0.6391404271125793, -2.9793379306793213, -0.02578870952129364, 1.9190301895141602, 0.7607088685035706, -0.6439675688743591, 0.0345151461660862, 2.051497459411621, 0.8561277985572815, 0.5663543939590454, -1.1014809608459473, -0.12230177968740463, 0.5829750299453735, 0.4135463237762451, -0.9320701956748962, 1.8791346549987793, 0.08525310456752777, -1.0915433168411255, 1.3731786012649536, 1.3564841747283936, -0.7330632209777832, -0.6766387820243835, 0.3507261574268341, 0.04885122552514076, -0.17338497936725616, 0.8632403612136841, 0.47879523038864136, -1.4257055521011353, -0.675957202911377, 2.0632400512695312, -2.4170100688934326, -1.2903708219528198, 0.4949870705604553, 0.3645097017288208, 0.21755175292491913, 0.6203315258026123, 0.28331422805786133, -0.7763705849647522, 0.20031382143497467, 1.1696887016296387, -1.3695083856582642, 1.6495120525360107, -0.44445255398750305, -0.7701156139373779, 1.3954051733016968, -0.17573095858097076, -1.1101011037826538, 0.989737868309021, -1.1640733480453491, 0.26526567339897156, -1.5291736125946045], [0.7569635510444641, 1.6744868755340576, 1.0104416608810425, -0.40064942836761475, 0.4719146490097046, 0.9801095724105835, 0.8724814057350159, -0.5484398007392883, -0.08091799914836884, -0.570275604724884, -0.8117756247520447, -0.8153510689735413, 0.0438646599650383, -0.10091952979564667, 1.919770359992981, 0.5643632411956787, 1.8952512741088867, 0.27320966124534607, 0.38976097106933594, 0.3704376518726349, -1.4901180267333984, 0.9290827512741089, 0.5386424660682678, -2.6337380409240723, -0.22097715735435486, 2.026590585708618, -0.6734390258789062, -0.23185239732265472, 1.0497257709503174, 0.008994543924927711, 0.7243160009384155, 1.042382001876831, 0.7431557178497314, -1.9197531938552856, 1.375734806060791, 0.5154474973678589, -0.5552363395690918, -0.9265751838684082, -0.24519729614257812, 0.11424969881772995, -1.311203956604004, 1.5816316604614258, -0.9002759456634521, -0.14647097885608673, -0.5978180766105652, -0.45890191197395325, -0.591010332107544, -0.38115987181663513, 0.5870726108551025, -0.08276399970054626, -0.052097491919994354, 0.4160902500152588, 0.7537116408348083, -1.4522498846054077, 0.4355459213256836, -0.0024808694142848253, 0.3238249719142914, 0.8222692608833313, 0.49990010261535645, -0.8554880619049072, 1.0055699348449707, -0.2695976793766022, -0.8177878260612488, 0.5132935047149658, 0.11161833256483078, -1.761256217956543, -0.5422370433807373, 0.9026356935501099, 0.3473820090293884, -0.7617737054824829, -0.9343904256820679, -0.06841355562210083, -1.3709609508514404, 0.836746096611023, -0.8619173765182495, 0.2527461051940918, -0.6321061849594116, 0.40684548020362854, -1.778035283088684, 1.8202091455459595, -0.23571966588497162, 1.4718947410583496, -1.939184308052063, -2.50842022895813, 0.6057426333427429, 0.9897405505180359, 1.299870252609253, -1.1382805109024048, 1.4574342966079712, 1.4393401145935059, 1.3335641622543335, 0.5584449768066406, 0.9529978632926941, -0.33632105588912964, 1.4529982805252075, 0.9490828514099121, -0.19781935214996338, -0.3272883892059326, -0.7688826322555542, -0.10955631732940674, 0.9575356841087341, 1.4373455047607422, -1.1794918775558472, -0.06959175318479538, 0.506388247013092, -0.24068139493465424, -0.8776005506515503, -0.9018012285232544, 1.0598193407058716, -0.17975196242332458, 0.8785749673843384, -0.7876264452934265, 0.5665019750595093, 1.8528441190719604, -1.2370227575302124, 0.35067665576934814, -0.2347724586725235, -0.9874962568283081, -1.9801054000854492, 1.226850986480713, -1.5937309265136719, 0.5589422583580017, 0.5536475777626038, 0.6047728061676025, 0.37849730253219604, -1.3362457752227783, -0.22610992193222046, 1.686111569404602], [-0.8296763300895691, 1.299092173576355, 0.023450519889593124, 2.2684924602508545, -1.8208844661712646, 1.321446418762207, -1.0851025581359863, 2.412292003631592, 1.6029059886932373, -0.9235089421272278, 0.3443416953086853, 1.4007606506347656, 0.9001098871231079, -0.4338648319244385, 0.426104873418808, 0.2044752687215805, -0.8640519380569458, 0.8994717597961426, 0.21017968654632568, -2.702364444732666, 0.48007693886756897, -0.2256038933992386, -0.3047283887863159, -0.09248249977827072, 1.0460435152053833, 1.535447359085083, 0.3550828695297241, 0.7568509578704834, -0.3026500642299652, -0.8962259888648987, 1.6905542612075806, -1.0963395833969116, 1.5996490716934204, -0.8650141358375549, 0.5527515411376953, -1.6913059949874878, 0.32096582651138306, -0.33987152576446533, 0.9517778158187866, 0.3477936089038849, 1.5583134889602661, 0.5975027084350586, -0.09466376900672913, -0.2620641589164734, 0.056092388927936554, -1.118801474571228, -0.4721527099609375, -0.6366707682609558, -1.7311275005340576, 0.5162289142608643, 1.2699027061462402, -0.2025098353624344, -0.560046911239624, -0.09499455988407135, -1.1471325159072876, -1.578073263168335, -0.5088286399841309, -0.6901755928993225, 0.22925278544425964, -0.0357673317193985, 0.8800845742225647, -0.9513397812843323, -0.818599283695221, -0.9123839735984802, -0.32335329055786133, 0.04842546954751015, 1.4620287418365479, -1.0002751350402832, -0.5134477615356445, 1.473652958869934, 0.528596043586731, 1.3380475044250488, -0.7016807198524475, -0.49074044823646545, 0.5699030160903931, -2.2792437076568604, -0.16411210596561432, -0.5369028449058533, -1.3049578666687012, -0.22942601144313812, 0.9660118222236633, 0.39459455013275146, -0.09666141122579575, 0.5954211354255676, -0.5530166029930115, -0.5872302055358887, -0.5150029063224792, -2.24981951713562, 0.391353040933609, 1.1574994325637817, -0.8869112133979797, 0.7042246460914612, -1.2006078958511353, 1.081562876701355, -1.423938274383545, 0.5982099771499634, -0.6852119565010071, 0.9638488292694092, -1.063525915145874, 0.7822751402854919, 0.48454728722572327, 1.705183982849121, 0.2010224312543869, 0.12091057002544403, 0.3948741555213928, -0.10308749228715897, 1.6527167558670044, 0.3792537748813629, 1.019055724143982, -0.9768548011779785, 1.358742117881775, 0.5829475522041321, -1.2518277168273926, -0.19746321439743042, 0.07742343097925186, 0.34282657504081726, 1.3387268781661987, -0.4080294966697693, -0.5939579010009766, -0.20532841980457306, 0.12440922856330872, 0.24435310065746307, 0.910283625125885, -1.8836323022842407, 1.3609881401062012, -0.42415741086006165, -0.9066532850265503, -0.46436259150505066], [-0.5491686463356018, 1.2485069036483765, 1.1763005256652832, -1.663278341293335, -0.09951432794332504, 1.0302355289459229, -0.3873363435268402, -0.30462485551834106, -1.8121278285980225, 1.0544352531433105, 1.4778467416763306, 0.6788762211799622, 1.2334901094436646, -1.1307337284088135, -0.31426793336868286, 0.6514925360679626, -1.841810941696167, -0.07880917936563492, -0.0367463082075119, 0.7068172097206116, -1.3639646768569946, 0.08242647349834442, 1.311833143234253, -1.2237493991851807, 0.3302477300167084, -0.11389598995447159, -0.526603639125824, -0.6664672493934631, 1.2022349834442139, -0.23047009110450745, -0.4449574947357178, -2.0824315547943115, -0.6660215854644775, -0.14212319254875183, -0.22314049303531647, -0.09589216858148575, -1.1804943084716797, -0.9490644335746765, -1.4962153434753418, -0.5937564969062805, -0.2616441547870636, 0.46602991223335266, -0.4643418490886688, 0.8147707581520081, 0.7385210394859314, -1.1443486213684082, 0.3549870550632477, -0.395283043384552, -0.2076861709356308, 0.9077290892601013, -1.4950335025787354, -1.5051671266555786, -1.8425240516662598, -0.514955997467041, -1.161754846572876, -0.8123315572738647, 0.30297666788101196, 1.106826901435852, 0.19849999248981476, 1.680418848991394, 0.5365053415298462, -0.10917573422193527, -0.45456835627555847, 0.6944599151611328, 1.1764851808547974, -0.6453471183776855, -0.33656349778175354, 1.722755789756775, -2.3116376399993896, -0.6627458930015564, -0.18244194984436035, -0.2694690227508545, -0.7495408654212952, -0.7055478096008301, -0.6171051263809204, 0.2985900640487671, 0.3411477208137512, 0.37455862760543823, 1.0364149808883667, -0.6318444013595581, 0.7708097100257874, 0.8738576769828796, 0.6613374352455139, 0.7536190748214722, 0.7076880931854248, 1.53812575340271, 2.685128927230835, 2.4664320945739746, 1.05569326877594, 0.41915905475616455, 0.9543363451957703, 0.1405208259820938, 1.3574961423873901, 1.9276736974716187, 0.21949626505374908, -0.7369973063468933, -1.1829183101654053, 0.22616879642009735, 0.08053962886333466, 0.0005576914991252124, 0.3403608202934265, -0.7134871482849121, 1.4966133832931519, 0.1783970445394516, -0.6353947520256042, 1.597917914390564, -0.5102381706237793, -0.5802053809165955, -0.22277700901031494, 0.18953128159046173, 0.3816494643688202, 2.498485565185547, -0.24000878632068634, 0.9368844032287598, -0.4304213225841522, -0.29889774322509766, 1.270642876625061, -0.793293297290802, -1.3069875240325928, 1.897606372833252, 0.8880518674850464, -1.6958467960357666, -2.1862287521362305, -0.1142747551202774, -1.1170344352722168, -1.1634587049484253, 0.03964382782578468, 0.8266208171844482], [0.1703444868326187, -1.0421407222747803, 1.2475143671035767, -2.0010457038879395, 0.530653715133667, -0.8644345998764038, -0.5831479430198669, 0.39772841334342957, 1.3085217475891113, 0.016055790707468987, -0.2140871286392212, -0.15505573153495789, 0.587440013885498, -1.1059976816177368, 0.5394435524940491, -2.8764963150024414, 2.2405123710632324, -1.6364375352859497, -0.8604456782341003, 0.741022527217865, 1.6631004810333252, -1.21802818775177, 1.7764172554016113, 0.8098073601722717, 0.5007696151733398, 1.0546643733978271, -0.7276665568351746, -1.707724690437317, -0.6217606067657471, -0.7737146019935608, 0.2070731222629547, -0.28713056445121765, -0.9741770029067993, -0.10289083421230316, -0.8496991395950317, -0.2719458341598511, 0.12492981553077698, 0.03429212421178818, 1.001533031463623, -0.7929449677467346, 0.9789808988571167, -0.19052381813526154, -0.9051229953765869, -0.07551658898591995, -0.4066540002822876, 0.6658419370651245, -1.8172560930252075, -0.2360650897026062, 0.45441749691963196, 0.07270772755146027, -0.018855946138501167, -0.09245336800813675, 0.08667929470539093, -0.8907976746559143, 0.5817063450813293, -0.07495324313640594, -0.5974228382110596, 1.6773790121078491, 0.6496627926826477, -0.24921156466007233, -0.41990575194358826, 0.2351502925157547, -0.5315492749214172, -0.9399434924125671, 2.0066721439361572, 0.2178904265165329, 0.08140698075294495, -0.18166813254356384, 0.8618612289428711, 0.364918053150177, -0.23125888407230377, -0.6418948769569397, -1.0103942155838013, 0.9614297747612, 0.4821423888206482, 0.9489249587059021, 1.5380281209945679, 2.059246063232422, -0.2166927307844162, -2.08966064453125, 0.5235106348991394, -0.5068898797035217, 1.411775827407837, 0.9802467823028564, -0.6223383545875549, 0.3455028831958771, -0.41050413250923157, -0.4817357659339905, -1.1524696350097656, -1.6266785860061646, 0.3028411865234375, 0.010144662111997604, -0.8548511862754822, 0.42205363512039185, -2.332242488861084, 3.7106635570526123, -0.2379009872674942, 0.3339063823223114, 0.9394364356994629, 0.6404125094413757, 2.0142898559570312, 0.3471863865852356, -0.6864508986473083, -0.3705769181251526, -0.13175058364868164, 1.3397417068481445, 0.4963512420654297, 0.33304205536842346, 2.0069942474365234, -0.1523188054561615, -1.3957469463348389, -0.6972726583480835, -0.27434036135673523, -0.11518184095621109, -0.7021275162696838, -1.7242083549499512, -0.7357265949249268, 0.2568766474723816, 1.029660940170288, 0.49581554532051086, -0.7808252573013306, 0.713943600654602, 1.3400863409042358, 1.240054726600647, -0.14533056318759918, 1.2271345853805542, 0.14246800541877747, 0.46787330508232117], [-0.4878605604171753, -1.7439287900924683, -0.5720717310905457, 0.017997711896896362, -0.30679699778556824, -0.07599279284477234, 0.027347952127456665, 0.5930087566375732, -1.41312575340271, -1.1964045763015747, 0.9007646441459656, 1.072189450263977, -0.3549748361110687, -0.7820167541503906, 0.8639442920684814, -1.3953285217285156, -0.5088895559310913, 0.5501572489738464, 0.2391640692949295, -1.305361270904541, 1.0930808782577515, 0.3867863714694977, -0.0061734868213534355, -0.4851137399673462, 0.027169523760676384, 0.657261073589325, -2.2887589931488037, 1.1945455074310303, -0.8481540083885193, -1.9554768800735474, -0.6421894431114197, -0.731655478477478, -0.5033467411994934, 0.8764767050743103, 0.2986384630203247, -0.8629484176635742, -0.5704033970832825, -0.11688321083784103, 0.10687076300382614, 0.553970456123352, 0.6213012337684631, 1.8515595197677612, -0.590307354927063, -1.7274055480957031, 0.28109052777290344, -0.06749150156974792, -0.19579094648361206, -0.7850112915039062, -1.4092594385147095, 0.8539255857467651, -1.5475982427597046, -0.8673751950263977, -0.13368377089500427, -0.5333600640296936, 1.3034801483154297, 0.3603108823299408, 0.48249977827072144, -1.006432056427002, -0.02460944466292858, -0.3224646747112274, -0.4953984022140503, -0.5490997433662415, 0.7501201629638672, -0.17444685101509094, -0.37061938643455505, -1.7301431894302368, -0.18789413571357727, 1.3297576904296875, -1.0401873588562012, 0.4268922507762909, -1.097198247909546, 0.511532187461853, 0.41466236114501953, -0.44646215438842773, -1.4394338130950928, 0.5970536470413208, -1.5386042594909668, -0.5419512391090393, -0.6186660528182983, 0.5476309061050415, -0.667857825756073, -1.0949515104293823, -0.07660363614559174, -0.17190083861351013, 0.615574061870575, -0.5228646397590637, -0.6294432878494263, -1.8395074605941772, -0.8278981447219849, 1.013939619064331, -0.5545334219932556, -0.9013041853904724, -0.3304517865180969, 0.38851502537727356, 0.6762593388557434, 0.8244448304176331, 0.16595739126205444, -0.39090240001678467, 1.5134479999542236, 1.775022268295288, -1.6413847208023071, -1.119648814201355, 0.18083983659744263, -0.37536969780921936, 1.6028904914855957, 1.2132209539413452, -0.7757843136787415, 0.47696977853775024, 0.631287693977356, 0.37101590633392334, -1.4272061586380005, -0.3411843180656433, 0.15719923377037048, 0.2324392944574356, 1.8308380842208862, 0.374653160572052, 0.33692604303359985, 0.1799536645412445, 0.29196253418922424, 2.600576162338257, -1.8141028881072998, 0.866870641708374, 0.16480223834514618, 0.48909077048301697, 1.6326303482055664, -0.004586884286254644, 0.6013807654380798, -0.3763686418533325], [-0.9105314016342163, -0.20090317726135254, -2.2822725772857666, -0.6520428657531738, 2.6676247119903564, -0.3063388466835022, 0.042621299624443054, -0.4634534418582916, 1.2872107028961182, 0.01639050617814064, -0.22439107298851013, -0.5356351137161255, 0.33504074811935425, 1.8314182758331299, -1.248130202293396, -1.6963036060333252, 1.7054487466812134, -0.057548221200704575, 0.311568945646286, -0.8131557106971741, -0.8667059540748596, 1.4208165407180786, 0.8896965980529785, 0.22273565828800201, -1.000063180923462, -0.08023601770401001, 0.10054569691419601, -1.133890151977539, 0.05475988611578941, 0.38972315192222595, -0.27938443422317505, -2.1024112701416016, 0.8200177550315857, 0.6542035341262817, 1.012110710144043, -0.37852928042411804, -0.8287041783332825, 1.6252518892288208, 0.03210775554180145, -1.8204210996627808, 0.38275885581970215, 0.6539356708526611, -1.9845402240753174, -0.9216435551643372, 1.338104009628296, 0.7407345175743103, 0.21674342453479767, 1.2140445709228516, -1.3679888248443604, 0.382909893989563, -0.9918320178985596, 0.35911527276039124, 0.8240230679512024, -1.2741198539733887, -0.706084132194519, -1.6616569757461548, -0.20861893892288208, 0.5739856362342834, 2.1527884006500244, -0.1729069948196411, 0.19985182583332062, 0.15454675257205963, -0.5466386675834656, -0.7898863554000854, 0.36605316400527954, -0.6588950157165527, -0.866274356842041, -0.9922454953193665, -2.4328091144561768, 0.7612747550010681, -0.26360222697257996, -0.15819144248962402, 0.0105897793546319, -0.033449023962020874, 1.7756896018981934, 0.5241246223449707, 0.7932764887809753, 1.389523983001709, 0.907732367515564, -0.7121052742004395, 0.47485432028770447, 0.13155902922153473, -0.35495832562446594, 1.4801887273788452, 0.5130997896194458, -1.5814062356948853, 0.3831281065940857, 0.5723833441734314, 0.031051672995090485, -0.4750472903251648, -0.7904250621795654, -2.4411604404449463, -1.2543905973434448, -1.83176851272583, -0.1649453341960907, 0.7121037840843201, -0.9927001595497131, -0.24186807870864868, -0.8405930995941162, -1.2479506731033325, 0.7169595956802368, -1.254926323890686, 0.6291861534118652, -0.8010694980621338, 1.079454779624939, 0.6587393283843994, -0.20239105820655823, 0.8158583641052246, -0.43512651324272156, -0.5600654482841492, 0.6340155601501465, -1.8306291103363037, 0.9524387717247009, -0.9081577062606812, -1.864478588104248, 0.1240483745932579, -2.162412643432617, 0.23401102423667908, 0.6443668603897095, 0.9136126637458801, 2.0322225093841553, -0.06388382613658905, -2.3669838905334473, -0.9693589210510254, -0.38578158617019653, -0.9747700095176697, -1.5042078495025635, 0.3231208920478821], [-1.152817964553833, 0.7121239900588989, -0.9793249368667603, 1.8496240377426147, -0.9445121884346008, 0.30623894929885864, -1.9668132066726685, -0.19022150337696075, 1.4908150434494019, -0.07630638778209686, -1.6277387142181396, -0.5454502105712891, 0.7187318801879883, -0.5187021493911743, -0.6929386258125305, -2.0525271892547607, 0.9939894676208496, 2.2156052589416504, -0.06775761395692825, 1.5704946517944336, -0.20300406217575073, -0.864690363407135, -1.5950424671173096, 1.4425008296966553, 2.045652389526367, -0.44283729791641235, -0.18021267652511597, -1.1146634817123413, -0.34815895557403564, 0.5012924671173096, 0.1266603320837021, 1.3025380373001099, 2.15263032913208, 0.15563447773456573, -1.6588184833526611, 0.7292454242706299, 0.2123962938785553, -1.294884204864502, -1.0828800201416016, 1.626692533493042, 0.14699523150920868, 1.5081946849822998, -0.2764599025249481, 0.4439254105091095, 0.9694486856460571, -0.7257580161094666, 1.2026137113571167, 1.65218186378479, -0.2849387228488922, 1.6271504163742065, 0.5624379515647888, -1.2820067405700684, -0.5730717778205872, -0.762447714805603, -0.16986727714538574, -0.7613399624824524, -0.3198355734348297, 0.09981473535299301, 0.975123941898346, 1.9646133184432983, -0.9795780181884766, 1.3552422523498535, 2.6077747344970703, -0.16593536734580994, -1.1763999462127686, -1.5601171255111694, 1.5562852621078491, 0.05644337460398674, 1.1055082082748413, 0.38992151618003845, 0.6588464975357056, -0.33817943930625916, -1.202960729598999, -0.11097630858421326, -1.1092109680175781, -0.965462327003479, 2.0255186557769775, -1.6976429224014282, -0.2317267805337906, 0.5012519359588623, 1.142270803451538, -0.35231226682662964, 1.717118263244629, -0.3368302881717682, 0.32102686166763306, -2.186328172683716, 1.4462060928344727, -1.3482598066329956, -0.44251832365989685, -0.7713744640350342, -0.7227358818054199, -0.2885909080505371, -0.7025549411773682, 1.3094241619110107, 0.09748751670122147, 0.7398956418037415, -1.3348817825317383, 1.5125749111175537, 0.9183720946311951, 0.07487102597951889, 0.4445728063583374, 1.4842075109481812, -0.8151668906211853, -1.950803518295288, -0.7985820174217224, 0.06685107946395874, -0.5239350199699402, -1.0632070302963257, 0.09377104789018631, 0.7636808753013611, 1.5307995080947876, -0.43172261118888855, -0.02481011301279068, -0.5292566418647766, -2.242922067642212, 0.6149752736091614, -1.0286492109298706, -0.5299652814865112, 0.38857680559158325, -0.35995909571647644, -0.1886904537677765, -0.5886325836181641, 1.162929892539978, -0.2805352210998535, -1.0144529342651367, -2.10853910446167, -2.4208624362945557, 0.24210944771766663], [-0.6118433475494385, -0.6058788895606995, -0.6841298341751099, 1.7087311744689941, 0.31417664885520935, -1.9228532314300537, 1.4040738344192505, -0.9340675473213196, -0.06286679953336716, 0.8079544305801392, 0.11966830492019653, 2.014174222946167, 0.2850154936313629, -0.8826680779457092, 1.2166675329208374, -0.775115430355072, -0.30323711037635803, -1.38799250125885, 0.49774661660194397, 0.04298209398984909, 1.6493749618530273, -0.6426688432693481, 0.5394780039787292, 1.297857642173767, 0.2511897087097168, -1.6187447309494019, -2.2208943367004395, -2.132160186767578, -1.6388967037200928, 0.18228428065776825, 0.14882349967956543, 0.07071448862552643, -0.3079290986061096, -0.3212459087371826, 1.0652964115142822, -0.9209237694740295, 2.676279067993164, 0.7516860961914062, 0.9333329796791077, -1.7287665605545044, -1.7267616987228394, 0.23549455404281616, -0.3749028444290161, 0.39754927158355713, -0.40853074193000793, 0.9535879492759705, 0.23324444890022278, 0.8871498703956604, 0.10231286287307739, 0.23056219518184662, 0.7127149105072021, 0.4195939600467682, 0.769970178604126, 0.6270360946655273, -0.6205242872238159, -0.5906449556350708, -0.8434113264083862, -0.6381627917289734, -0.5078111290931702, -1.8735883235931396, -0.5435094237327576, -0.8229668140411377, -1.1229877471923828, -0.7490007877349854, -1.969103455543518, -0.3032089173793793, -0.2849541902542114, -0.37262022495269775, -0.4704088866710663, -1.598703384399414, 0.2233370840549469, -1.1272982358932495, 0.5730206966400146, 0.9709693789482117, 1.4051454067230225, 0.8882649540901184, -2.524448871612549, 2.356354236602783, 0.5453757643699646, 0.24002040922641754, 0.1222224086523056, 1.5009489059448242, -1.2982707023620605, -1.5651583671569824, 1.1655765771865845, 1.6086095571517944, 0.4529702663421631, 0.5678185224533081, 1.7679551839828491, 1.5002540349960327, -1.3104450702667236, -0.7391625046730042, 1.4914978742599487, 0.5922343730926514, 0.8567760586738586, 0.007955264300107956, -0.0727362334728241, 1.0626318454742432, -0.46118175983428955, 0.15908312797546387, -1.2590303421020508, -1.8421000242233276, -0.1888212412595749, -0.17288847267627716, 1.6810047626495361, 0.46830618381500244, 0.5937699675559998, -0.6318779587745667, 0.7232053875923157, 0.21243907511234283, 0.21978022158145905, 0.7804718613624573, -1.5223804712295532, -0.5020652413368225, -0.5589131116867065, 0.5395838022232056, -0.41445788741111755, -0.4158981442451477, -1.0340325832366943, -0.2146865874528885, 0.5128827691078186, 0.20234262943267822, -0.6963644623756409, 0.5476828813552856, 0.7170664072036743, 0.33271393179893494, -0.3923124670982361, 1.4288455247879028], [-0.2031133621931076, -1.715464472770691, -1.7810416221618652, 0.642498791217804, -0.6655020713806152, 1.799794316291809, -1.8137314319610596, 1.5011179447174072, 1.1368581056594849, 1.40720534324646, 0.20240147411823273, 1.8473138809204102, -0.0544152669608593, -0.43545040488243103, 0.9586518406867981, 0.4316682517528534, 0.6863041520118713, -0.8301643133163452, 0.5683351159095764, 0.17507661879062653, 0.16867896914482117, -0.8879278302192688, -0.07023946195840836, 0.47958648204803467, 1.59109628200531, 1.072843313217163, 0.7515246868133545, 0.3202343285083771, 0.827012300491333, -0.035874154418706894, -0.6735265254974365, 0.7202804088592529, 0.27182120084762573, 0.3784357011318207, -0.9817094802856445, -0.31520840525627136, 0.3548576831817627, -0.22309203445911407, 0.0561923086643219, 0.9432722926139832, -0.5116296410560608, 0.3720754086971283, 0.3489193618297577, -0.20175214111804962, -2.2317795753479004, 0.5166912078857422, -0.39105215668678284, -0.4218495190143585, 1.003834843635559, 0.033204879611730576, -0.24535533785820007, 1.0327610969543457, 0.7761799693107605, -0.47522038221359253, 0.1849210262298584, -0.7943980693817139, -0.5747599601745605, 1.3474812507629395, -0.7925856113433838, -0.6721049547195435, -0.37752774357795715, 0.3386014699935913, 0.9112295508384705, -0.3329279124736786, 1.5328017473220825, -0.9256547689437866, 1.0910948514938354, 0.7786988019943237, 0.05652662366628647, -0.8536621928215027, 0.6361204385757446, 1.8813241720199585, -1.3447926044464111, -0.22348976135253906, 0.7800334692001343, 0.12739849090576172, -0.9886268377304077, -2.622385263442993, -1.964718222618103, 0.29966914653778076, 0.3131829798221588, -0.7101679444313049, -0.24743607640266418, 0.8943252563476562, 1.1480494737625122, -0.08274425566196442, -0.06155785545706749, -2.0417890548706055, -1.2540878057479858, 0.684843122959137, -0.7988213300704956, 0.9919675588607788, -0.41369155049324036, -0.7353371977806091, -0.24788054823875427, 1.0665405988693237, 0.49388617277145386, -1.319715976715088, -0.05400783568620682, -0.12598799169063568, -0.39060601592063904, -0.9473543763160706, -0.5973961353302002, 0.9385213255882263, -0.8471829891204834, -0.07485691457986832, -0.32283815741539, 0.40603187680244446, -0.4270251989364624, 0.7710287570953369, 0.6739598512649536, -0.26987358927726746, 0.46363136172294617, 0.9506042003631592, 0.11000145226716995, 0.23485329747200012, -1.8164347410202026, 0.5788387656211853, -0.694950520992279, 0.27682924270629883, -1.9295233488082886, 1.1204758882522583, 0.7885861992835999, 0.4245152175426483, 1.3464351892471313, 1.1869999170303345, 0.14677174389362335, -0.6943272948265076], [1.037593126296997, 1.359885573387146, -1.0420479774475098, -0.766249418258667, -1.179237723350525, -0.34704115986824036, -0.7496975660324097, -1.6205554008483887, 0.5365379452705383, -0.9022252559661865, 1.5292514562606812, 0.5766884684562683, -0.13513007760047913, 0.7111337780952454, 2.3242762088775635, -0.7970829606056213, -1.3063427209854126, -0.11705004423856735, 0.2558038532733917, 0.8438387513160706, 0.6107924580574036, 0.3462335467338562, -0.5173033475875854, 1.3917566537857056, 0.9448468089103699, -0.2194366753101349, 0.38720861077308655, -0.8194106221199036, -0.1527930051088333, 0.9424098134040833, 1.4902033805847168, 0.7096362113952637, -1.1006709337234497, -0.42587634921073914, 0.6086241006851196, -1.5334855318069458, -2.0921733379364014, 0.43259453773498535, 1.0822110176086426, 2.7113664150238037, 1.5087459087371826, 0.36600878834724426, -1.9312493801116943, -1.6355575323104858, 1.318080186843872, -0.7098495960235596, -1.1264803409576416, 0.12634196877479553, -1.1373229026794434, 0.25217878818511963, -0.8516297936439514, -0.25038617849349976, -2.228759765625, -0.556468665599823, -0.4294205605983734, -1.3413772583007812, 0.9324784278869629, -1.911824107170105, 0.9500294923782349, 0.6509804725646973, 0.11776456981897354, -0.693302571773529, -0.39239004254341125, 1.6800079345703125, -1.0540772676467896, -0.696898341178894, 0.3953584134578705, -0.9087831377983093, -0.6093488931655884, 0.7678235173225403, 0.658776581287384, -0.9036016464233398, 1.5860131978988647, -0.33554771542549133, 0.4300939440727234, -0.7487747073173523, 1.3359150886535645, 2.3140361309051514, 2.1377828121185303, 0.003169435076415539, 0.026961615309119225, 0.6993094086647034, -1.1436365842819214, -1.9987964630126953, 1.6225353479385376, -0.07280930131673813, -0.5333480834960938, 1.1948482990264893, -0.331461101770401, 0.8047428727149963, 0.776978611946106, -0.6483163237571716, 1.502239465713501, 1.1350098848342896, -0.5361555218696594, -0.3638506233692169, 1.6284680366516113, 0.5358354449272156, 1.27030611038208, 0.433001309633255, -0.8031932711601257, 0.44511663913726807, -0.38059207797050476, -0.13730350136756897, -0.5564236640930176, 0.39557158946990967, -0.0010935653699561954, -1.336909532546997, 1.2294011116027832, 2.2162294387817383, 0.47931012511253357, 0.7551690340042114, -0.3934750556945801, 1.1222485303878784, -0.47616100311279297, -1.1142616271972656, 1.3326467275619507, 0.748676061630249, -1.3312702178955078, -1.4788497686386108, 0.23508530855178833, 0.24087005853652954, -2.030667543411255, 0.8801771998405457, 0.8543329834938049, 0.08398047834634781, 0.8750160336494446, 1.6555202007293701], [-0.24521033465862274, -0.4598110616207123, -2.327113389968872, 1.4809727668762207, -0.8794218301773071, 1.0599967241287231, -1.4167073965072632, 2.3351852893829346, 1.0073710680007935, 0.26825714111328125, 0.9808425903320312, -1.7323836088180542, 2.6578726768493652, -0.5221295356750488, -0.39704468846321106, 1.1404554843902588, 0.8685501217842102, 0.4941859543323517, -1.1606634855270386, -1.7508543729782104, 0.29367581009864807, -0.48432841897010803, -0.8121354579925537, 0.8564830422401428, -0.8117498755455017, 0.3495715260505676, 1.9690736532211304, -0.18587395548820496, -0.35201555490493774, 0.9690021276473999, -0.04805208742618561, 0.5133786797523499, 0.7922642230987549, 1.4660601615905762, 0.6384323835372925, -0.9698423743247986, 0.9350240230560303, 1.8108397722244263, 1.1622087955474854, 0.3712543845176697, -0.6814975142478943, 1.67984938621521, -0.5387322306632996, 0.12224645912647247, -0.43324586749076843, 0.6027276515960693, 0.2309790551662445, 0.21149548888206482, -1.327663779258728, -0.8690558671951294, -0.5311740636825562, 0.8329710960388184, 0.8704860210418701, 0.4168357849121094, 0.49360883235931396, 0.060155075043439865, -0.6761301755905151, -0.015836821869015694, -0.9061166048049927, 0.017732134088873863, -0.25036105513572693, 0.25000086426734924, -0.07879388332366943, 0.778153657913208, -1.116320013999939, 0.5844866037368774, -0.3142145276069641, -0.968712329864502, 1.3995801210403442, -1.3984990119934082, 0.044220853596925735, 1.6627088785171509, -1.8813457489013672, -0.7706892490386963, -0.09237031638622284, -1.2415226697921753, 1.5012613534927368, 1.956005573272705, -1.393769383430481, -1.3939757347106934, 0.40506142377853394, -0.8503071069717407, 0.9452424645423889, -0.15660181641578674, 1.5255982875823975, 1.8235706090927124, 0.39946866035461426, -0.7877188920974731, 0.0011566767934709787, -2.303999185562134, 0.660001277923584, -1.5092594623565674, -0.1828344762325287, 1.6267428398132324, -1.3792892694473267, 0.37346744537353516, -1.3435035943984985, -0.8653835654258728, 0.16578561067581177, 1.161827802658081, -2.5105741024017334, -1.198278784751892, -0.0993141457438469, -0.761955976486206, -0.3813022971153259, 0.4482186734676361, 1.3697813749313354, -0.020679855719208717, 1.0688668489456177, -0.6234690546989441, 1.0425828695297241, -1.092887043952942, -0.32872316241264343, -0.6699380278587341, 1.253564476966858, -1.8148589134216309, -1.8228631019592285, 0.5765841603279114, -0.4539690315723419, 0.44402721524238586, 2.1862330436706543, 0.8103130459785461, -0.5830438733100891, 0.43714287877082825, -0.8104554414749146, -1.3415427207946777, -1.21769118309021, -1.0583670139312744], [-0.18821264803409576, -2.4321963787078857, -0.30376335978507996, 0.3671421408653259, 1.244457483291626, -0.08187976479530334, -0.04228805750608444, -1.464524507522583, 2.297720193862915, 0.4989206790924072, 1.1758209466934204, -0.8869984745979309, 0.7800905704498291, -0.03415989130735397, -0.7738083004951477, -0.8740308880805969, -0.6744934320449829, -0.8093104958534241, 0.7306948900222778, -0.17997321486473083, 2.0948596000671387, 0.3110545575618744, 1.0605417490005493, -0.1752137541770935, -1.1145187616348267, 0.041338589042425156, -0.15569862723350525, -0.865635097026825, 1.3614821434020996, -0.6321187019348145, -0.8358040452003479, -0.35383617877960205, 0.14496253430843353, 1.1800918579101562, 1.5604918003082275, 0.1041029840707779, -0.7111517786979675, -0.1646192967891693, 0.639728844165802, -1.1514493227005005, 2.79848313331604, -0.6732430458068848, 0.8102198839187622, -1.075445532798767, -2.013913631439209, 0.07816876471042633, 1.0671063661575317, 0.5544818043708801, 0.15462106466293335, -0.48469799757003784, -1.4833165407180786, 0.040034063160419464, -0.2897835969924927, -0.4626242518424988, 0.3276630640029907, -0.10294710099697113, -0.05492227151989937, 1.6736857891082764, 1.5282212495803833, -1.4152626991271973, -1.2233774662017822, 1.1973662376403809, -1.8518563508987427, 2.251065969467163, 0.2783076763153076, 1.4395294189453125, 1.6026651859283447, -1.0518689155578613, 1.5192731618881226, -0.0910818800330162, -1.2114650011062622, -0.5258087515830994, -1.739113688468933, 0.9642657041549683, -0.04074260592460632, 0.731732964515686, 0.8860049247741699, 0.5671272873878479, -1.182747483253479, -0.7637284398078918, -1.2150789499282837, 0.06404846161603928, 0.9949413537979126, 0.7904939651489258, 0.3638865053653717, 1.9467710256576538, -0.0009654454188421369, -1.2820760011672974, -0.74250727891922, -1.1054260730743408, 1.1650972366333008, -0.029407275840640068, -0.4933509826660156, -1.1654012203216553, -0.10590212047100067, -0.7393977046012878, -0.3108924925327301, 1.5745586156845093, -0.5211952924728394, 0.21006067097187042, -0.8196840286254883, -0.5754159092903137, 0.08431075513362885, -1.2085040807724, -0.10596194863319397, 2.9222044944763184, 1.3121421337127686, 1.127195954322815, 0.5231081247329712, 0.6358593106269836, 1.785069465637207, -1.6718649864196777, -0.3736518919467926, -0.6712822318077087, -0.05796971172094345, -0.07800305634737015, 1.9612371921539307, 1.0305830240249634, -1.0629925727844238, 2.31268310546875, -0.17715111374855042, -1.285822868347168, -0.09418027102947235, 0.7899071574211121, 0.115531787276268, -1.641632080078125, -1.9302895069122314, 2.010249137878418], [-0.6522538065910339, -0.16943466663360596, -0.7327204942703247, 0.2112041562795639, 0.09726274013519287, -0.5331650376319885, -0.39857399463653564, 1.9209550619125366, 1.151015281677246, -0.8844631314277649, 0.7063306570053101, -0.533198893070221, 1.6677721738815308, 1.5169661045074463, -0.10169188678264618, 0.02443292923271656, -0.9739500284194946, -0.17668521404266357, -0.6579224467277527, -1.043614149093628, -1.601672887802124, -0.34392502903938293, 0.12690038979053497, -1.8622933626174927, 1.8547863960266113, -0.5209785103797913, 1.0283796787261963, 0.21322448551654816, -1.1170622110366821, -0.9194915294647217, 0.4098207354545593, -0.059307556599378586, 1.4061017036437988, 0.7350631952285767, 0.48450884222984314, 0.1687297374010086, 0.7928224802017212, 0.7105991244316101, 0.3901059329509735, 2.0794312953948975, -0.8897309303283691, 0.1402883380651474, 1.120633602142334, 0.8343303799629211, 0.25222480297088623, 1.1095809936523438, 1.5877782106399536, -0.20023266971111298, 0.26854774355888367, 1.2611627578735352, -0.45903366804122925, -0.6906383633613586, 1.672622561454773, -0.4052915573120117, -0.43794941902160645, -1.8658132553100586, 1.1405956745147705, -0.6957376599311829, -2.4779670238494873, -0.7520730495452881, -0.0622791163623333, 0.4701215624809265, -1.5611523389816284, 0.6748583316802979, 0.11475759744644165, 1.8141894340515137, 0.696165919303894, 1.2681218385696411, -0.2612064480781555, -0.006940032355487347, -0.033901575952768326, -0.7653824090957642, -1.7915594577789307, 0.4503415524959564, 0.38530290126800537, 0.8659111857414246, 1.2290921211242676, -0.23831535875797272, 1.7501094341278076, -1.070114016532898, 0.11541265994310379, 0.09854072332382202, 0.041610874235630035, -0.43464988470077515, -0.5853456258773804, 0.7560644149780273, -1.1068962812423706, -1.5909546613693237, 0.2593856453895569, -1.2580289840698242, 1.2786848545074463, 1.3423457145690918, -0.06408030539751053, -0.25018468499183655, -0.18840545415878296, -0.7018937468528748, -0.4931681752204895, -0.2230161428451538, -0.2279050350189209, -1.589764952659607, -0.4168524742126465, 0.42399418354034424, 1.4426449537277222, -1.5709028244018555, -0.8798632621765137, 0.697615385055542, 0.8885865211486816, -0.32771024107933044, -0.3407801687717438, 0.011963393539190292, 1.6100802421569824, 0.12350554019212723, -0.313086599111557, -1.0802325010299683, 0.9352281093597412, 1.4449743032455444, 0.20708270370960236, -0.7877798676490784, -1.0046762228012085, -0.40663978457450867, 1.1136562824249268, 0.6942866444587708, -0.14708653092384338, -0.3347582221031189, 0.04500102251768112, 1.6271729469299316, -0.1649291217327118, -0.26334771513938904], [-0.8217685222625732, 0.9453756809234619, -0.10805945843458176, 0.6104508638381958, 1.263696551322937, 0.10041876882314682, -0.11887454241514206, 1.331400990486145, -0.7129884362220764, -1.848183274269104, 0.17245064675807953, -1.0567277669906616, 0.3952493965625763, 0.010805203579366207, 1.9369136095046997, -1.026047706604004, 1.7045903205871582, -2.7333619594573975, -0.43433693051338196, -2.507951498031616, 0.03648177161812782, 0.17497535049915314, 0.7347514033317566, -0.34013015031814575, 0.43568962812423706, -0.6859201192855835, -0.9696358442306519, 0.9470031261444092, 1.0935497283935547, 0.588288426399231, -1.0546960830688477, 1.285019040107727, -0.24296869337558746, -0.6180821657180786, 0.11190158128738403, -0.6835626363754272, -1.2808716297149658, -0.9788998365402222, -0.34934452176094055, -1.0250887870788574, -1.0012037754058838, -0.46224460005760193, -0.1471937745809555, 1.2836995124816895, 1.4436243772506714, 1.1424251794815063, -0.9386257529258728, 0.5624061822891235, -0.20890167355537415, 0.42883166670799255, 1.0408813953399658, -0.309980571269989, -1.3193966150283813, -0.9714531302452087, 1.4345026016235352, 0.5545352697372437, -1.3236925601959229, -1.6468826532363892, 2.2580208778381348, 1.4885412454605103, 0.6039423942565918, 0.232587993144989, 0.4305546283721924, -0.1563902050256729, -0.7061384916305542, -0.2537039816379547, -0.9166035056114197, 1.5550827980041504, 1.3992692232131958, 0.9522679448127747, -0.8543938994407654, -0.45486965775489807, -0.24079273641109467, 1.1177120208740234, 0.46873900294303894, -0.11457560211420059, 1.0176560878753662, -0.9184947609901428, -0.5683987736701965, 0.564835250377655, 0.1528194546699524, -2.457206964492798, -1.242165446281433, 1.089112639427185, -0.3433130979537964, -0.0038441673386842012, -1.776776909828186, 0.24629560112953186, 1.60441255569458, 1.4113603830337524, 1.0984551906585693, 0.07685159146785736, -1.260240912437439, 0.35968708992004395, -0.27494052052497864, 1.009390115737915, -0.7517189979553223, 0.9147520065307617, 1.626779317855835, -0.2952643930912018, -0.022953474894165993, -0.4203304350376129, -1.2850561141967773, 1.7706308364868164, 0.48785659670829773, 0.6401118040084839, -0.9915173053741455, 1.8610844612121582, -0.024371765553951263, 1.6462482213974, 1.303301215171814, 1.326924443244934, -0.6454318761825562, 0.364125519990921, -0.1692298948764801, 1.199713945388794, -0.13996060192584991, -0.8569706082344055, 0.14000679552555084, 0.9698678255081177, -0.03778814524412155, 0.47583234310150146, 1.2489734888076782, 0.38767924904823303, -0.48691490292549133, 1.5345808267593384, -0.35030806064605713, 2.0265934467315674], [-0.43761542439460754, -0.014792338013648987, -1.376862645149231, 0.36164045333862305, 1.3657824993133545, -0.511661171913147, 0.4605236351490021, 0.3480198383331299, -0.11118793487548828, -0.8647215962409973, 0.18207699060440063, -0.6838656067848206, 0.8397624492645264, -0.34733012318611145, 0.11714063584804535, -1.6928671598434448, 0.24431119859218597, -0.6675083637237549, -0.6126616597175598, -1.5344080924987793, -0.3664148151874542, -0.7450488805770874, 0.29374000430107117, 0.9906485676765442, 0.08044610917568207, 0.17599739134311676, -0.6226302981376648, -0.06563811749219894, -0.5503016710281372, -0.007428525947034359, -0.1571034938097, -1.9031518697738647, -1.2201343774795532, -1.1933566331863403, -1.6372685432434082, -0.2641684412956238, -1.3367637395858765, 0.4717008173465729, -1.2090405225753784, -0.20499324798583984, -2.2930967807769775, 0.6527910828590393, -2.581070899963379, 0.3191966116428375, -0.6724383234977722, -0.8273330330848694, -0.9469701051712036, -0.795184850692749, 0.027267929166555405, 0.3835744559764862, 0.9731076955795288, -0.33314210176467896, 1.941713809967041, 0.19701670110225677, -0.10511398315429688, -2.07336688041687, 0.02718576416373253, 1.2470197677612305, 0.05373372882604599, 0.9580878019332886, -0.42152339220046997, -1.1129767894744873, -0.5180655717849731, 0.0168134868144989, 1.4140392541885376, 0.3943137526512146, -1.6657429933547974, -0.7566961050033569, -0.9668576717376709, 0.699731171131134, 0.5695874094963074, 0.15433257818222046, -2.35071063041687, -0.35880711674690247, -2.050384998321533, 1.709041953086853, 0.22821295261383057, 0.18293538689613342, 1.0462641716003418, -0.1302899718284607, -0.6169705390930176, 0.6150175333023071, 0.09112253785133362, -2.2503154277801514, 0.5262961387634277, -0.2623142600059509, 1.172029972076416, -0.4050358831882477, 0.6188161969184875, 0.22997719049453735, -1.9033135175704956, 1.5615224838256836, -1.8155019283294678, -1.7669330835342407, 2.1051228046417236, 0.08787012845277786, -0.1084488108754158, 0.1930125504732132, 1.6369630098342896, -0.6845299601554871, 0.10541276633739471, -0.095376156270504, 1.2393698692321777, 1.5559132099151611, -0.20267193019390106, 0.8889117240905762, -1.3498413562774658, -0.222185879945755, -0.8627582788467407, -0.6597787141799927, 0.0628959909081459, 1.2468141317367554, 0.7406957149505615, -1.0595444440841675, -0.44600340723991394, -2.03901743888855, 0.31022992730140686, -0.04639926552772522, -0.5237605571746826, -1.1018129587173462, 0.28639277815818787, 1.173557996749878, 1.2283018827438354, -0.30446577072143555, -0.5221477150917053, 1.788213849067688, 1.1249618530273438, 0.4194752275943756], [-0.41163045167922974, -0.5974428653717041, -0.5434810519218445, -0.15143324434757233, -1.1013965606689453, 0.9470228552818298, 0.13573342561721802, 0.698334813117981, -0.20145033299922943, 1.573253870010376, 1.086297869682312, -0.21653740108013153, -1.3582093715667725, 1.8081178665161133, -1.2509424686431885, -1.6125675439834595, 1.325749397277832, -0.9488911628723145, -0.09785857796669006, 1.6666148900985718, 0.3207959830760956, 1.8240097761154175, 0.28849366307258606, -1.6933902502059937, 0.6343601942062378, -0.4662655293941498, 1.2786451578140259, -0.46451011300086975, -0.2905012369155884, -1.0684475898742676, -0.5836449265480042, 0.08114629238843918, 0.5508427619934082, -0.19753220677375793, -0.6937586069107056, -0.40034034848213196, 0.8873762488365173, -0.004730222746729851, -1.4628850221633911, -0.0854986310005188, -0.8167746067047119, 1.6133062839508057, -0.7226545214653015, 0.503756582736969, -1.4473150968551636, -0.35883212089538574, -1.0387974977493286, -2.7267050743103027, -0.7142855525016785, -0.36701369285583496, -0.7143040299415588, -0.9717468023300171, -0.2072857916355133, 0.5210498571395874, 1.3188802003860474, -0.1323036253452301, -0.2256748527288437, -0.3145330846309662, -1.3333332538604736, -0.5956048965454102, 2.239572763442993, 0.462026983499527, -1.0270825624465942, 0.9939824938774109, -0.2645772397518158, 0.7574008107185364, -0.45309725403785706, -0.36688095331192017, -0.8765560388565063, -0.2661290764808655, 0.5335756540298462, 0.4956183135509491, -0.36990833282470703, -0.4077608585357666, -0.288239061832428, 0.21026071906089783, 0.4088630974292755, -0.28380700945854187, 0.8755524158477783, 0.33206701278686523, 0.1668885350227356, 0.3672214150428772, 1.603805661201477, 0.01734374277293682, 0.1836753636598587, -0.14351843297481537, 1.9480197429656982, 1.1832177639007568, 0.43661293387413025, 0.026486005634069443, 2.6796979904174805, 1.2691636085510254, -0.34138020873069763, 0.44741198420524597, 0.8761164546012878, -0.2154477834701538, 0.14765430986881256, -0.4327271282672882, -2.563267707824707, 0.38919591903686523, 0.09434101730585098, 1.6068147420883179, 0.8097140789031982, 1.1400566101074219, 0.5147802233695984, -0.5477482080459595, 0.6824995279312134, 0.6506572365760803, 0.9687690138816833, -0.23012462258338928, -1.483079195022583, -0.03727182000875473, 0.5226946473121643, -0.2156665325164795, 1.6126251220703125, -1.111256718635559, 0.0024230999406427145, 0.31541094183921814, -1.9033920764923096, 0.5872029662132263, -1.6149650812149048, -0.39917248487472534, -0.35015469789505005, -1.1528180837631226, -1.7907490730285645, 2.316532611846924, -0.37473833560943604, -1.1941235065460205], [0.14660094678401947, -0.9943248629570007, 1.077983021736145, -1.3462501764297485, -1.4859007596969604, 0.03050614334642887, 0.6690276861190796, -0.5053986310958862, -1.8175055980682373, 0.26394209265708923, -0.17666451632976532, -0.49872997403144836, 0.10962381958961487, -1.3414618968963623, -0.8430644273757935, 1.4314911365509033, 0.18091118335723877, 0.25102582573890686, 0.3992370069026947, 0.375230610370636, 0.0011632618261501193, -0.7446199059486389, -2.4559857845306396, 0.15970443189144135, 0.8260374665260315, -1.873087763786316, -0.7812329530715942, 0.033432379364967346, 1.0619608163833618, 1.0058741569519043, 0.38374975323677063, 0.8322835564613342, 0.7215616106987, -0.3557044565677643, -0.7661757469177246, -2.5840044021606445, -0.35330772399902344, 0.5374554991722107, 1.2227717638015747, 0.41320884227752686, -1.437935471534729, -0.19572220742702484, 0.39472153782844543, -0.6782954335212708, -0.5004286766052246, -1.7826812267303467, -0.5405934453010559, -1.254795789718628, 1.2092660665512085, 1.7615541219711304, -0.22771427035331726, -1.403225302696228, 0.213036447763443, -1.1578449010849, 0.8525311350822449, 1.9944111108779907, 1.1898375749588013, -0.6886191368103027, -0.24722355604171753, -0.1825227439403534, -0.8476340770721436, 0.3236966133117676, 0.04526655748486519, -0.28946247696876526, 0.19894354045391083, 1.4058504104614258, -0.2825111150741577, -0.16427059471607208, 0.024816062301397324, 0.552213728427887, 1.120011568069458, -0.1457691192626953, -1.3754873275756836, 0.3120931386947632, 1.1026610136032104, -0.8326013684272766, -0.36840447783470154, -0.4628343880176544, -1.4513579607009888, 0.08439585566520691, 0.3279498219490051, 0.950234591960907, 0.5673049092292786, -1.5149518251419067, -0.5190092921257019, -0.7339840531349182, 1.5389354228973389, -0.009493747726082802, 0.18415892124176025, 1.3696284294128418, -1.8402589559555054, 0.3583970069885254, -1.2462178468704224, -0.5895755887031555, -0.4936516582965851, -0.2546910047531128, 1.2147713899612427, 0.30411195755004883, 0.38059234619140625, -0.11972304433584213, -0.8344109654426575, 1.4749557971954346, 0.2723429501056671, -1.3048248291015625, -1.849221110343933, 0.8933733701705933, -0.9460949301719666, -0.19234158098697662, 0.4819498062133789, 0.16242265701293945, -0.4765366017818451, 0.7359886765480042, -0.12429752200841904, -1.4391682147979736, -1.0313560962677002, -0.3027993142604828, -0.5752386450767517, 0.001837314572185278, -1.1515967845916748, -0.9241822361946106, 1.1906198263168335, 0.7288256287574768, -0.27066054940223694, -0.9208565950393677, -1.4791545867919922, 1.23431396484375, 0.2566356360912323, 0.4110599160194397]]], [[[0.694597601890564, 1.6241843700408936, 0.271711528301239, 0.8427097201347351, 2.6756579875946045, -1.9542969465255737, 0.38914889097213745, -0.20871879160404205, -1.390931487083435, 0.8214238882064819, 0.4071633219718933, 0.47290417551994324, 0.6904619336128235, 0.9990369081497192, -0.14376047253608704, -0.2217620313167572, 1.3266035318374634, 2.4469234943389893, -0.31675949692726135, -0.1094452366232872, 1.4317902326583862, -1.5216563940048218, 0.2930331528186798, 1.2297708988189697, 1.0723649263381958, -0.21114031970500946, -0.7982872724533081, 0.7888241410255432, -1.2142044305801392, 0.34392574429512024, -1.2331520318984985, -0.123710997402668, 0.2390526980161667, -0.6554268598556519, -0.1073836162686348, -0.9479185342788696, -0.5875685811042786, -0.09348037838935852, -0.00675663398578763, 1.9564380645751953, 0.3272824287414551, -0.08159850537776947, 0.6819081902503967, 0.7540727853775024, -1.3884588479995728, -0.18423695862293243, -0.3524768352508545, 0.5867161154747009, -0.8385539650917053, -1.4485805034637451, 1.1110316514968872, 0.3494747579097748, 1.332360029220581, 1.3255491256713867, -0.5148575901985168, -0.0443541519343853, -2.6855976581573486, -1.6164923906326294, -0.3101355731487274, 0.8099266290664673, -0.29828551411628723, -1.2894301414489746, -0.45951950550079346, 0.8941766023635864, 0.6796748042106628, -2.0100808143615723, 0.37919721007347107, 0.43443092703819275, -0.7802673578262329, 0.4666885733604431, -1.0782032012939453, -0.3319030702114105, -2.124166250228882, 0.2705543339252472, 0.6211799383163452, 0.9241507649421692, 1.863283395767212, -2.1955926418304443, 0.019335005432367325, 0.020697053521871567, 0.7772040963172913, -1.6268959045410156, 0.7732356190681458, -0.04902360588312149, -1.3600696325302124, -0.18251453340053558, -0.2815423011779785, 0.4019791781902313, -0.5832284688949585, -1.0539376735687256, 0.1622142642736435, -1.4327815771102905, -0.36811327934265137, 1.2753114700317383, -2.386746883392334, -0.7086278200149536, -0.8557430505752563, 1.4815666675567627, 0.25732576847076416, 0.8529709577560425, -0.6580915451049805, 0.03188977390527725, 1.7419861555099487, 1.320378303527832, -1.2601590156555176, -1.1412785053253174, 0.2817009687423706, 0.4772505462169647, 1.5121445655822754, -1.295815348625183, -0.29891499876976013, 0.8306915760040283, -1.31504487991333, 1.8706961870193481, -0.5617458820343018, 0.6457124352455139, -0.021569564938545227, -0.13501453399658203, -1.1485093832015991, 0.8995501399040222, -0.9102432131767273, -0.24036730825901031, 0.697221577167511, 0.5184691548347473, -0.2739841043949127, -1.6952487230300903, 0.033343516290187836, 1.5663540363311768], [0.365113765001297, 2.7541375160217285, 1.1650841236114502, -0.5700231790542603, 1.047511100769043, -0.14610803127288818, 0.08176425844430923, 0.5618574023246765, -0.6810012459754944, 1.392762303352356, -0.5005173087120056, 0.6470136046409607, 1.5359094142913818, 0.04591406509280205, 0.5710014700889587, 0.2204350382089615, 0.9594249725341797, -0.5094887018203735, 0.383994460105896, 0.11522045731544495, -0.7885671854019165, -0.43682125210762024, 0.5603554844856262, 0.07671641558408737, 1.599774718284607, 0.07854816317558289, -1.1026737689971924, -2.1710357666015625, 0.21405884623527527, -0.5471674203872681, 1.1074074506759644, 2.0255770683288574, -0.736802339553833, -0.34846198558807373, 0.5439313054084778, -1.2315257787704468, -1.101053237915039, 0.6101252436637878, -0.7055027484893799, 0.06506949663162231, 0.5007779598236084, 1.112064003944397, -0.6224310398101807, -0.9899448156356812, 0.28705549240112305, 0.04574573040008545, 0.6523086428642273, 0.9162456393241882, -0.2928774952888489, 0.5844024419784546, 0.263195276260376, 0.2338249832391739, 0.6354791522026062, 1.300286054611206, 1.467141032218933, 1.2126678228378296, 0.1713566780090332, -0.08755464106798172, -0.47568926215171814, -1.171004295349121, 0.1865834891796112, -0.4971311688423157, -1.9044467210769653, -0.8355363607406616, -1.402133822441101, -1.306223750114441, -1.2182706594467163, 1.3512215614318848, -0.1704508662223816, -0.2849326431751251, 0.15542493760585785, 0.9270729422569275, -0.8466202020645142, -1.4296529293060303, -0.8787830471992493, 0.09170734137296677, 0.5675197839736938, -1.022139549255371, 0.5729379653930664, -1.2522484064102173, 0.8990680575370789, 1.6411616802215576, -0.8114784359931946, 0.577465295791626, 1.4545142650604248, 0.6723246574401855, 1.064023733139038, -1.2059568166732788, -1.00542414188385, -1.6704350709915161, 1.3274660110473633, 0.3783557415008545, -1.4180787801742554, 0.05229923874139786, -0.05259767174720764, -2.2601068019866943, -0.36448222398757935, -0.25092801451683044, 0.6357272267341614, -1.4711405038833618, 0.8286300301551819, -0.45943930745124817, -1.516377329826355, -0.33007705211639404, 0.09146931022405624, 0.013345059007406235, -0.04592134431004524, -0.9080505967140198, 0.7951149344444275, -0.39231038093566895, -1.0861608982086182, 1.4005829095840454, -0.6998178362846375, 0.5228323340415955, -0.09623589366674423, 0.9927820563316345, 0.7148724794387817, 0.2520262897014618, 0.9897408485412598, -0.5987343192100525, 0.650937557220459, 0.4527132511138916, -0.6523313522338867, 0.1440502107143402, 0.6414507031440735, 0.7546187043190002, 0.8095828294754028, -0.18528752028942108], [-0.6885858774185181, 0.1576734334230423, -0.3492845594882965, -0.648830771446228, -0.8629646301269531, 0.34245991706848145, -0.2058829814195633, -1.981561303138733, -1.8036218881607056, -1.2530232667922974, 1.4254162311553955, -0.6172107458114624, -2.062326669692993, 0.9393829703330994, -0.3318127691745758, 1.1789113283157349, -0.37260007858276367, -0.7914496064186096, 1.0712591409683228, 1.0945429801940918, -0.6546511054039001, -0.8679757714271545, -0.32317474484443665, -0.49724987149238586, -1.5865927934646606, 0.5607123374938965, -0.16403955221176147, -1.053196668624878, -0.7564137578010559, -0.9378822445869446, -1.7148340940475464, -0.06928329169750214, 1.512778401374817, -1.1813396215438843, -0.32118725776672363, 0.520592987537384, 1.1418036222457886, -1.6788922548294067, -0.2474033236503601, 0.6354559063911438, 1.5387098789215088, -1.2897669076919556, 0.058183956891298294, 0.7506427764892578, -0.4436027407646179, -1.5621920824050903, -1.1267703771591187, -0.39598825573921204, -2.7786812782287598, -0.18605628609657288, -0.7124753594398499, -0.794702410697937, -0.061875373125076294, -1.4126102924346924, -1.3962368965148926, 0.6075954437255859, -1.2000092267990112, 0.9577990770339966, -1.6809055805206299, 0.9517614841461182, -0.5516961812973022, -2.6227948665618896, -0.24358417093753815, -0.7847016453742981, -1.0602258443832397, -1.8657475709915161, 2.258964776992798, -0.9862587451934814, 0.670644223690033, -0.4163394272327423, -0.18751533329486847, -0.42384079098701477, 0.5526183843612671, 0.7613446712493896, 0.09999853372573853, 0.45544981956481934, -1.1457542181015015, -1.1815729141235352, 0.5391883254051208, 0.26988986134529114, 0.038884665817022324, -1.555314540863037, 0.13799796998500824, -1.003246545791626, -1.5212889909744263, 0.0493612177670002, 0.4616903066635132, 0.9995251297950745, -0.1237037181854248, -1.3682416677474976, -1.381839394569397, -1.1636441946029663, 1.9402620792388916, -0.2120586484670639, 0.6012899279594421, -0.7928561568260193, 0.4643881916999817, 0.5445193648338318, 1.4147388935089111, -2.2697255611419678, -0.7133532166481018, 0.8994521498680115, 1.5975289344787598, 0.03792396932840347, -0.989917516708374, -0.4297875761985779, 0.4674808382987976, 1.8871002197265625, 2.4781880378723145, -0.5480957627296448, 0.027899151667952538, 0.5014910697937012, -0.5462809205055237, 0.21356341242790222, 0.3249742090702057, 0.3418107032775879, -0.5826072096824646, -0.1653284728527069, 0.786207914352417, 0.09208457916975021, 0.9025844931602478, -0.4158799350261688, 1.3001445531845093, -1.6411033868789673, -0.4281521439552307, 0.6538912057876587, -0.48124754428863525, 0.05564876273274422], [0.02448185905814171, 0.7677097320556641, 0.26869940757751465, -1.4666343927383423, 1.9175870418548584, 1.1470410823822021, 0.152822807431221, -1.9727234840393066, -0.19732129573822021, 2.2085704803466797, -0.11826065182685852, 0.6673054099082947, -0.9700974225997925, 1.729263186454773, -0.8369856476783752, -1.0565041303634644, 2.484126567840576, -0.0990760549902916, -0.6774198412895203, 1.879263997077942, 0.14911508560180664, -0.9336154460906982, -0.20501790940761566, -1.9578266143798828, 0.03228577598929405, -1.559158205986023, -0.004404398612678051, -1.8468267917633057, -1.9449619054794312, 1.4515190124511719, 0.504618227481842, 0.6184650659561157, 0.7530920505523682, -1.4359194040298462, 0.7730253338813782, -1.2758432626724243, 1.1068278551101685, -0.2436511218547821, 0.7450934052467346, -0.10851166397333145, 2.3700101375579834, 0.7995666861534119, 0.873540997505188, -1.4536727666854858, -0.8156836032867432, 1.2001917362213135, -0.6597049236297607, -1.919579267501831, -1.5316541194915771, -0.2587011158466339, 0.344975084066391, 0.1884002536535263, 0.7742927074432373, 0.7761949896812439, -0.49903908371925354, -1.2268904447555542, 0.1824672371149063, 2.3196470737457275, -1.8283438682556152, -0.6333131790161133, 1.104933261871338, -0.7367445826530457, -2.2096309661865234, 1.934537410736084, -0.3936825096607208, -0.6121600270271301, -0.04794308915734291, -0.8556787371635437, 1.9567739963531494, -1.4465528726577759, -2.693067789077759, 0.45730265974998474, 0.40506047010421753, 0.7989001274108887, -0.7487321496009827, -0.5310860872268677, -0.005169682204723358, 0.00326637364923954, 0.13390837609767914, -0.18229778110980988, -0.267024964094162, 1.239484429359436, 0.5324646830558777, 0.28525474667549133, -0.8302850723266602, 0.12121812254190445, 0.8517482280731201, 0.80357825756073, 0.28388965129852295, 1.884385347366333, 1.0003454685211182, -1.7947735786437988, -2.227803945541382, -0.604864239692688, -1.6965556144714355, 1.7483528852462769, -2.0130221843719482, -0.5012568235397339, 1.5502758026123047, -1.6536643505096436, 2.6653478145599365, 0.8285954594612122, -1.8272652626037598, -0.38949841260910034, -0.1460680514574051, -0.6411862969398499, -0.0842522606253624, -1.2879188060760498, 1.2869062423706055, 0.20751415193080902, -2.484445333480835, -2.1124706268310547, 3.413768768310547, -0.21612393856048584, 0.41099613904953003, 0.39466744661331177, -0.4437045753002167, 1.1963775157928467, 0.7443804144859314, 2.2082929611206055, -1.356050968170166, 0.8141420483589172, -0.6279206275939941, 0.7632437944412231, -1.790385127067566, 1.8243355751037598, 0.5235496759414673, 0.317815363407135], [-0.6476888060569763, 0.20583568513393402, -0.4824412167072296, 0.9329496026039124, -1.0190455913543701, -0.36923864483833313, 1.457061529159546, -0.5910245180130005, -0.23972371220588684, -1.2734668254852295, 1.0263334512710571, -1.2271816730499268, 1.6978275775909424, 1.5343087911605835, -0.04403449222445488, 1.0108144283294678, 0.5544885396957397, -0.527109682559967, -0.33197134733200073, -0.9391604661941528, 2.189840793609619, -0.4159608483314514, 1.3110244274139404, 0.24286538362503052, -0.12075001001358032, -0.3609735071659088, -1.604457139968872, 0.4096547067165375, -0.4593172073364258, -0.4878008961677551, -0.4776344299316406, 0.4086533486843109, 0.5276384949684143, 1.471174955368042, -0.1808050125837326, -0.47031018137931824, 1.1040666103363037, -1.7539860010147095, 1.4561145305633545, 0.07228093594312668, 0.3337249755859375, -1.8139485120773315, 0.20832936465740204, -0.10571163147687912, -0.8625559210777283, 1.150819182395935, -0.35216742753982544, -1.311265230178833, 0.20800241827964783, 1.726298213005066, -0.6199381947517395, -0.6057175993919373, -0.010347818024456501, 1.064064621925354, 0.7738332152366638, -1.2469494342803955, -0.528974711894989, -1.1401748657226562, 0.6498213410377502, -0.7022775411605835, 0.0885644182562828, -0.03675062954425812, 0.8241100907325745, 0.2637859582901001, -0.1307871788740158, 0.9160415530204773, -0.921876847743988, -0.90997713804245, -0.6318694353103638, -1.033129096031189, 0.6136009097099304, 0.9343653321266174, -0.522525429725647, 0.17260988056659698, -1.3886620998382568, 0.3242170214653015, -0.4710122346878052, -0.7772091627120972, 0.012462170794606209, 0.2187185287475586, -0.017706453800201416, -0.24035967886447906, 0.6014532446861267, -1.331370234489441, 0.8812956809997559, -0.606950044631958, 0.10951083898544312, 0.9038297533988953, -1.2388027906417847, 0.8889403939247131, -0.5162791013717651, -0.1078273355960846, -1.0367401838302612, -0.23330016434192657, 0.5871246457099915, -1.5018866062164307, -0.32162266969680786, 0.537152111530304, 0.19899792969226837, -0.8965430855751038, -0.13917571306228638, -0.6746553778648376, 0.3081928491592407, 1.2764663696289062, 0.11549052596092224, -1.664250373840332, 0.37249529361724854, 0.2634139955043793, -1.3273674249649048, 2.063229560852051, -1.01571786403656, -1.9555250406265259, -1.1526318788528442, -0.5696362853050232, -0.18210245668888092, 0.600472092628479, 0.028232352808117867, -0.5800712704658508, -0.7277379631996155, 0.18438895046710968, -1.7765709161758423, 0.34426918625831604, -0.1037227213382721, 0.18440841138362885, 0.5416735410690308, 0.46816790103912354, 0.09317536652088165, -1.090173363685608], [0.9128084778785706, 0.17644083499908447, 0.3414241075515747, 1.3949897289276123, -0.09916940331459045, -0.30561718344688416, 0.9318339824676514, -0.515544056892395, -0.19161419570446014, 0.6006439328193665, 1.650509238243103, 1.629753828048706, 2.0188658237457275, 0.8094847202301025, 0.5447234511375427, -1.991302728652954, 0.4988342821598053, -0.25789812207221985, 1.1249468326568604, -0.16985104978084564, 1.3428064584732056, -2.3913094997406006, -1.8026180267333984, 0.4746532142162323, -1.132466197013855, -1.1170734167099, -0.6996747851371765, -0.1058412417769432, -0.7789146304130554, -1.0617740154266357, -0.9793086051940918, -0.0280990581959486, 0.8647139668464661, -0.12088757753372192, -0.8887237310409546, -1.130652904510498, 0.13536466658115387, 1.5329679250717163, 0.15823876857757568, 2.0400733947753906, -0.9097766876220703, 3.228903293609619, 0.9031848907470703, 0.006323200650513172, 1.5150010585784912, 0.8985671997070312, 0.30887725949287415, 1.0561765432357788, 0.40208739042282104, 0.6917154788970947, 1.7193281650543213, -0.3414320647716522, -0.6449424624443054, 0.6726080775260925, -1.0639504194259644, 0.761633574962616, 1.784532070159912, -0.5552530288696289, 0.03206481784582138, -1.221671223640442, -0.13775630295276642, -1.0888679027557373, 3.303589105606079, 0.3665515184402466, 1.1761595010757446, 0.22777439653873444, -1.2483007907867432, -0.03696794807910919, -0.08794622123241425, 0.3642462491989136, -0.8514370918273926, 0.029666727408766747, -0.3208983242511749, 0.08582472056150436, -0.742844820022583, 0.0044226376339793205, 0.6905057430267334, -0.902776300907135, 2.83182430267334, -0.05825558677315712, 0.10719742625951767, -0.5116783976554871, 0.29624009132385254, 1.397982120513916, 1.2376877069473267, -1.5993273258209229, 0.5331620573997498, -1.2683157920837402, 0.26179662346839905, 0.5388706922531128, 0.8144378066062927, 0.6608511209487915, -0.8398016691207886, 0.1560155600309372, 1.229217290878296, 0.3182811141014099, 0.4879171848297119, -0.2406652867794037, 0.08076322078704834, 0.7434536814689636, -0.3174993693828583, 0.1388251930475235, 0.4337250292301178, 1.1729881763458252, -0.592949628829956, 1.447239875793457, -0.24084384739398956, -1.782473087310791, -0.06203274801373482, 0.30960121750831604, 0.4792580008506775, 0.5233916640281677, 0.7247862815856934, 0.8224000334739685, -0.17614249885082245, 0.10105282068252563, 1.1182689666748047, -0.9819470643997192, -1.918686032295227, -0.6945884823799133, 0.04341096803545952, 0.29337361454963684, 1.9643968343734741, 1.4123435020446777, 0.8859716653823853, 0.737770676612854, -0.469499796628952, -1.7575591802597046], [0.6765257716178894, 0.183353453874588, -0.5500777363777161, 1.4439027309417725, 1.6711981296539307, 0.5853780508041382, -0.33381521701812744, -1.4294648170471191, 0.023982945829629898, -0.9132371544837952, 1.0903552770614624, 1.1916530132293701, 0.09672325849533081, 0.1348971724510193, 0.6726164817810059, -0.26156163215637207, -1.114250659942627, 1.4083659648895264, 0.3697601854801178, -1.8868191242218018, -0.8045341372489929, 2.405689239501953, 0.13483141362667084, 1.168255090713501, -0.14949320256710052, -0.5269834399223328, 0.5856959819793701, 2.44640851020813, -0.6483399868011475, -1.0697005987167358, -0.021027056500315666, 0.9336798191070557, -0.6348645687103271, -0.026622407138347626, -0.8249268531799316, -0.9021540880203247, 0.22492378950119019, -0.4350849390029907, -0.3352920413017273, -0.42687785625457764, -1.0761276483535767, -1.5404889583587646, 0.7090396285057068, -0.9667379260063171, 1.264998435974121, -0.1400068998336792, 1.1341263055801392, -0.7854750156402588, -0.5871216654777527, -1.034458875656128, -0.8360263109207153, 1.8778631687164307, -0.47063663601875305, 1.0226033926010132, 0.876922070980072, 1.4012534618377686, -0.5696443319320679, -0.28219908475875854, 1.678260087966919, -0.13849714398384094, 0.0472259521484375, 0.48374202847480774, -2.9499526023864746, 0.8053758144378662, 2.665315628051758, -0.7567940354347229, -0.5513619184494019, 0.6166907548904419, 0.44282281398773193, 0.5104397535324097, -0.9740613102912903, -0.7256649732589722, 1.663432002067566, -1.7516648769378662, -0.7113449573516846, -0.30955958366394043, -0.5234830379486084, 0.4464018940925598, -0.8339254856109619, -0.5951151847839355, -0.2000766396522522, -0.2835247814655304, -0.13717491924762726, 1.2601919174194336, 1.5229759216308594, 2.5322697162628174, 0.5784055590629578, 1.5982325077056885, 0.13979466259479523, 0.35138577222824097, -0.4868674576282501, -0.9821734428405762, -0.5790908932685852, 0.31319594383239746, -1.203905701637268, 1.422532320022583, -4.820175647735596, 0.16138969361782074, -0.7456915974617004, -0.9702907204627991, 0.16261029243469238, 0.4561690092086792, -0.08138522505760193, 0.2690717875957489, 0.16383200883865356, 0.6691933274269104, -0.12873294949531555, -0.11867298185825348, 0.11793386936187744, -1.9027374982833862, -1.8959755897521973, 0.1732272505760193, 0.22842136025428772, 1.2057465314865112, -0.9998956322669983, -0.31799861788749695, 0.8704087734222412, -0.7780575752258301, 1.0583184957504272, 0.03292539715766907, 0.35230395197868347, -0.4762890934944153, -0.31279081106185913, 0.6780009865760803, 0.9783734083175659, 0.24701239168643951, -0.46862971782684326, 0.6859331727027893], [0.745937705039978, 0.02526259422302246, 1.126860499382019, 0.7698380947113037, 1.0042908191680908, 1.054681658744812, 0.09062433242797852, 1.0338680744171143, -0.03737514466047287, 1.2435319423675537, -0.7686936855316162, -0.02897353656589985, -0.5655792355537415, -0.8254125118255615, -3.2538578510284424, -0.9616493582725525, -1.5559948682785034, -0.160612091422081, -1.4414799213409424, 1.515695333480835, -0.12992356717586517, 0.8789687156677246, -0.6993891000747681, 0.01329335942864418, -0.008344980888068676, -0.6041834354400635, 1.4935340881347656, 0.46901440620422363, -0.3323078155517578, 0.5094171762466431, 0.14115135371685028, -0.8608770966529846, 2.0694403648376465, 1.8251607418060303, 1.8465538024902344, -0.06382500380277634, -1.7833517789840698, 0.9265088438987732, -0.5580200552940369, -1.3221054077148438, 0.05983983352780342, 0.5509008169174194, -0.011265551671385765, -1.3969316482543945, -0.4037136137485504, -0.12961626052856445, 0.5229390263557434, 0.6187868714332581, 0.8326910734176636, -0.41101714968681335, -0.22406595945358276, -0.08056062459945679, -1.9480705261230469, -0.7482137680053711, -0.562956690788269, 0.3778020739555359, -0.02460491470992565, -1.9563674926757812, -0.383686900138855, 0.8866170048713684, -0.3839203119277954, 0.8167117238044739, 1.3645858764648438, 0.9405843615531921, 0.6217637658119202, 0.5243932604789734, 1.0919774770736694, -0.11330369114875793, -0.7856388092041016, 0.14367996156215668, 0.4706508219242096, 0.043990593403577805, 0.14945384860038757, 0.12016648799180984, -0.34977519512176514, -0.3841700255870819, -0.4797309637069702, -1.9085888862609863, 2.4058310985565186, 1.1312024593353271, -0.9974510669708252, -0.9464389085769653, -0.07238688319921494, 0.2882467210292816, -0.5237963795661926, 0.8887166380882263, 0.5898932814598083, 0.061291761696338654, -0.45149022340774536, 1.4956358671188354, -2.5721144676208496, 1.6086556911468506, -0.5446403622627258, -1.5192233324050903, -0.7116296887397766, 1.331295132637024, 0.8295690417289734, -1.184801697731018, -1.5039576292037964, -0.7995752096176147, 0.4139895439147949, 0.05239930748939514, -0.4119311273097992, -0.9282578229904175, 0.612138032913208, 1.6187089681625366, 0.8177744746208191, 2.9425878524780273, -0.6066650152206421, 1.024686336517334, -0.5676119923591614, 1.3843895196914673, -0.017566276714205742, 0.8546543717384338, -0.41224735975265503, 0.7038244009017944, -1.0565388202667236, -0.6482229232788086, -1.6920634508132935, -0.6247544884681702, 1.5892177820205688, -0.8576380014419556, -1.062958836555481, -0.7235020995140076, -0.34170836210250854, 1.9726510047912598, 0.2369910627603531, 1.1269488334655762], [0.44871073961257935, 0.21648700535297394, 1.5183604955673218, -0.8659560680389404, 1.7301867008209229, -0.7239362597465515, -0.6477221846580505, 1.407391905784607, -0.5969987511634827, -0.5912616848945618, 1.3485054969787598, -1.739894151687622, -1.3390944004058838, 1.1706682443618774, 0.5401647090911865, 1.9267566204071045, 2.4354867935180664, -0.8288998603820801, -0.6729724407196045, -1.0538939237594604, -0.7806921601295471, 0.6147233843803406, 0.4619240462779999, -0.7668566107749939, -0.6237510442733765, -1.1732409000396729, 0.3684265911579132, -1.5276775360107422, -0.838951826095581, -2.2637147903442383, -1.4296627044677734, -1.7095342874526978, 1.4447447061538696, -2.322061061859131, -0.5051980018615723, -0.32768714427948, -0.19208814203739166, 0.5397306680679321, 0.21156789362430573, 0.8232102394104004, -0.947512149810791, -0.0934741348028183, 1.7940967082977295, -0.05474310368299484, 0.09241490811109543, -0.9917853474617004, -1.3041669130325317, 0.44936898350715637, 0.7103468775749207, 0.9141449928283691, 0.9358986020088196, -0.6025363802909851, 0.422551691532135, -0.4311244487762451, -0.045242276042699814, -2.7442378997802734, 2.0200846195220947, 1.1067787408828735, 1.4601936340332031, -0.042931508272886276, -0.8528750538825989, -0.8644986152648926, 2.5943846702575684, 0.5146707892417908, -1.9018014669418335, -0.17117075622081757, -0.4492552578449249, 0.4912320375442505, 1.912423849105835, 1.2212145328521729, -0.45380502939224243, -0.08923421055078506, 0.20546500384807587, 1.006488561630249, 0.5090993642807007, -0.05160904675722122, -1.1840710639953613, 0.9803245663642883, 1.9649810791015625, -0.10011991858482361, -1.6843292713165283, 0.2905728816986084, 0.6310686469078064, 0.814713180065155, 0.9034900665283203, -0.06836636364459991, -1.2676876783370972, 1.0957138538360596, 0.33407580852508545, 1.1805758476257324, -0.853001058101654, -1.3942091464996338, 0.6333721280097961, 0.272551566362381, 0.6152641177177429, -0.4567748010158539, 1.3012579679489136, -0.3444361090660095, 1.1484854221343994, 0.8160215020179749, 0.5625500679016113, 1.4831291437149048, -0.06800162047147751, -0.5173318386077881, 0.45304709672927856, 1.6249141693115234, -0.09626076370477676, -0.5118141174316406, 0.008702577091753483, 0.9286964535713196, 0.42866089940071106, -1.7336794137954712, -1.5415457487106323, 0.7385653257369995, 0.0979432463645935, 1.650902509689331, -0.2787715792655945, 0.8551592826843262, 0.037656791508197784, -1.325850009918213, 0.7064269185066223, 0.20846788585186005, -0.05205410718917847, 1.2698440551757812, -0.5671555399894714, -0.01875930279493332, -0.286900132894516, 0.40904247760772705], [1.340644359588623, -1.355794072151184, -0.027303487062454224, -0.3982454538345337, -1.380478024482727, -0.08122152090072632, 2.6758430004119873, 0.5148954391479492, -0.2396203726530075, 0.5979204773902893, -1.6229151487350464, 0.4698674976825714, -1.9427390098571777, -1.286747932434082, -0.1723901927471161, -0.21629168093204498, 0.6797260642051697, 1.5572375059127808, -1.2500981092453003, -0.4107741117477417, -0.986835241317749, 0.3189694285392761, 1.1828124523162842, 0.2671050429344177, 1.7412410974502563, -0.17093078792095184, 0.3099488914012909, 0.3942311108112335, 0.5864620208740234, -0.568083643913269, -0.3637349307537079, 0.3482023775577545, 1.7247858047485352, 1.4142144918441772, 1.3860208988189697, -0.39428701996803284, 0.5237748622894287, -0.03294077888131142, 0.303300142288208, -0.7740530967712402, 0.41951626539230347, -0.45312637090682983, 0.5112583637237549, 0.9699077606201172, 0.8672553300857544, 0.4201951324939728, 1.734157681465149, -1.673262596130371, 0.08294888585805893, 1.4854841232299805, 0.5016087293624878, 1.7039172649383545, 0.946838915348053, 2.055453062057495, 0.0060079945251345634, -0.04395897313952446, -1.2356970310211182, -2.0640676021575928, -0.2651773989200592, 0.7348682284355164, 0.6720324754714966, -0.5884284973144531, -1.333330750465393, -0.11707667261362076, 1.0757766962051392, -0.8601413369178772, 0.49627405405044556, 1.1184402704238892, -0.8770818710327148, -0.3847719132900238, 2.0068724155426025, -0.21996307373046875, 0.4969042241573334, -0.8723759055137634, -0.7649458646774292, -0.4657686650753021, -1.2244489192962646, 1.2660642862319946, 2.420969247817993, -1.3764972686767578, 0.9573250412940979, 0.06681910902261734, -0.2471386045217514, -0.5816606879234314, 1.2072789669036865, -0.8397735357284546, 1.141706943511963, 0.8679011464118958, -2.30021071434021, -0.6142427325248718, 0.8078945875167847, -0.9551203846931458, -1.3604480028152466, -0.4405185878276825, -1.1037124395370483, 0.6524376273155212, -0.45613232254981995, -2.5641586780548096, -0.14324405789375305, 1.6839944124221802, 0.031356073915958405, -0.7211329340934753, -0.7097090482711792, 1.2916525602340698, 0.01208052784204483, -1.138242483139038, -0.09142931550741196, 0.28319084644317627, 1.3706121444702148, 1.7281602621078491, -1.1340057849884033, -0.8476801514625549, 1.4307681322097778, 0.8393747210502625, -0.7048782706260681, 0.8324342370033264, 0.3911930024623871, -1.3326040506362915, 0.1965458244085312, 0.9107168912887573, 0.7961984872817993, -0.1502142995595932, -1.290765404701233, -0.0614108182489872, -1.3402621746063232, 1.1443514823913574, 0.9404212236404419, 1.7248727083206177], [-1.0560365915298462, -0.4730237126350403, 0.7768740653991699, -0.46603405475616455, -2.127021551132202, 0.5573250651359558, 1.386657953262329, 0.0755394771695137, 0.0646592378616333, -0.01578443869948387, 1.353442668914795, -0.6549813747406006, -0.011421403847634792, -1.6865023374557495, -0.5455114841461182, 0.9014130234718323, 1.242668867111206, 1.0723968744277954, 1.4599674940109253, 1.4053857326507568, 0.4964004456996918, 0.04070070758461952, 0.7309662103652954, 0.9052167534828186, 2.524170160293579, 0.2973141372203827, -0.5990496277809143, 1.1493607759475708, 0.28896862268447876, -0.7028151750564575, -2.902761936187744, 0.8703272938728333, -0.1294102966785431, -0.037658143788576126, 0.5453738570213318, 1.7811055183410645, 0.36840271949768066, 0.2454691231250763, -0.41512539982795715, 0.8465649485588074, -0.666235625743866, 1.6222729682922363, 0.01841888576745987, 0.4527198076248169, 0.6736987829208374, -1.0766502618789673, -2.1340079307556152, 0.3654583692550659, 0.43920251727104187, 0.015577276237308979, 0.5466739535331726, 0.20866678655147552, -0.9172056913375854, 0.11838113516569138, -0.36402758955955505, -2.175936698913574, -0.530074954032898, 1.6198822259902954, -0.559438169002533, -1.2084085941314697, -0.04710075259208679, 0.259878933429718, -0.5267049670219421, 0.034638624638319016, 1.4160099029541016, -0.3909182846546173, -3.184718132019043, -1.4879939556121826, -1.4497522115707397, -0.3492499589920044, 1.1294407844543457, 1.1189380884170532, 0.24853552877902985, 1.2973936796188354, -0.005854939576238394, -0.6024107933044434, 1.7496553659439087, -1.6561769247055054, 0.2071857452392578, 1.343438982963562, 0.6042656302452087, 1.610970139503479, -0.5428681373596191, 1.4778789281845093, 0.859003484249115, 0.8280092477798462, -0.09811946004629135, -0.925775945186615, -1.1605912446975708, -0.8658491373062134, -0.603946328163147, 0.36895060539245605, 0.6842219829559326, 0.6598710417747498, -0.13912923634052277, 1.4343727827072144, -1.3246564865112305, -1.5401781797409058, -0.5036640763282776, 0.8316934108734131, -1.3039124011993408, 0.516282856464386, 0.9734742045402527, 1.3343952894210815, 0.63005131483078, 0.3685685992240906, -0.4693536162376404, -1.0003958940505981, -1.4009902477264404, 0.09581782668828964, 1.9824960231781006, 1.191725492477417, -1.240726351737976, 1.8429778814315796, 0.1735592931509018, 1.6166725158691406, -1.2886782884597778, 0.5398942232131958, -0.437678724527359, -0.5283066630363464, 0.3885490894317627, 0.35837996006011963, 0.011783872731029987, -0.15623153746128082, -1.4417495727539062, 0.9379849433898926, -0.47243696451187134, 0.714862585067749], [-2.6158435344696045, 0.1141253113746643, -0.2705560326576233, 1.191475749015808, 0.28474611043930054, 1.3056628704071045, -0.7390174269676208, -0.4440285265445709, -0.32966119050979614, -0.6692350506782532, 0.20311972498893738, -1.290342926979065, -0.1042570173740387, 1.1022356748580933, 1.1194323301315308, 0.016411835327744484, 0.9444718956947327, 0.3436449468135834, -1.4430615901947021, -0.7519723773002625, 0.897229015827179, -0.025139853358268738, -0.07673311233520508, -0.5219045281410217, 0.8570784330368042, -0.010053793899714947, -0.8433873057365417, 0.06016138568520546, -1.144208550453186, 0.11319153755903244, 0.04144900664687157, -1.0934021472930908, 0.19165945053100586, -0.31097137928009033, 0.463007390499115, -0.8773528337478638, 0.25877174735069275, -0.3056641221046448, -0.31121718883514404, 0.3108486831188202, 0.07945465296506882, 0.35278093814849854, 0.17849847674369812, -0.937484085559845, -1.092481255531311, -1.323617935180664, 0.9110068082809448, -0.2789793014526367, -0.735991895198822, 0.1387806236743927, 0.3947572708129883, 0.1018834263086319, -0.11071839928627014, 0.4092956781387329, -0.011348090134561062, -1.4730675220489502, 0.37726110219955444, -0.38755184412002563, -0.21701186895370483, -0.6579998731613159, -0.4895702600479126, -0.2989248037338257, -0.8807732462882996, -0.5291529893875122, 0.9226421117782593, 0.05609304830431938, 1.106062650680542, 1.7570440769195557, -1.2597724199295044, -0.8985150456428528, -0.11395715922117233, -0.05247143283486366, -0.4086087644100189, 0.02399991638958454, -1.8778421878814697, -0.8919371962547302, -0.5887132287025452, 0.884565532207489, -0.6950272917747498, 0.7731845378875732, 0.1574029177427292, 0.5671020746231079, 2.11812162399292, -0.028140820562839508, -0.44150981307029724, 0.9189707040786743, 0.4645043611526489, 1.4508788585662842, -1.3376542329788208, 0.176571786403656, 0.27558910846710205, 0.19155295193195343, -0.11091272532939911, -2.3469367027282715, 2.24043869972229, 0.6768949031829834, -0.21205033361911774, -0.46188169717788696, -0.440269410610199, 0.5336020588874817, 0.6422078609466553, -2.00974440574646, -0.5121827721595764, -0.05321747809648514, 0.9778660535812378, 1.2305952310562134, -1.5171246528625488, -0.10726901888847351, 0.285616934299469, 0.3684946298599243, 0.6932882070541382, 0.2070801705121994, -0.3894582688808441, -0.3864879608154297, -1.9241105318069458, 0.11928002536296844, -0.19798991084098816, -1.0535975694656372, -0.1207834780216217, -0.799866795539856, -0.0032044770196080208, 1.6023011207580566, 0.3434586822986603, -1.4450868368148804, 0.29201555252075195, -0.20602373778820038, -0.05448390543460846, 1.2550227642059326], [-0.059847719967365265, 0.6205161809921265, -0.4129248261451721, 1.2150133848190308, 1.6375315189361572, -1.1577855348587036, -0.6754137277603149, 0.5179288983345032, -1.1852084398269653, -0.0008872862672433257, -1.0071394443511963, -0.40954023599624634, -0.7976044416427612, -2.2049827575683594, 0.31807202100753784, 1.2590645551681519, 0.42096471786499023, -0.08648591488599777, 0.459638386964798, 0.8155697584152222, -0.7417249083518982, -0.23779986798763275, 0.1899319440126419, -0.3063620328903198, -0.19447241723537445, -0.7278440594673157, 0.021586978808045387, -2.0277199745178223, -0.40048378705978394, -2.236327648162842, 0.3985079526901245, -0.21129758656024933, -0.45135563611984253, -0.5093441009521484, 0.4826491177082062, -0.25532013177871704, 0.34352272748947144, -0.8297752737998962, 0.20523443818092346, 0.33328184485435486, 0.2501283884048462, -0.7505201101303101, 0.08776205778121948, -1.1962685585021973, 1.6246836185455322, 1.0549976825714111, 0.13058045506477356, 0.3376404345035553, -0.8533107042312622, -1.038838267326355, -0.5234265923500061, -0.18751712143421173, 0.4398159086704254, 2.1073484420776367, 0.9435060620307922, 0.7062227129936218, -0.5638723969459534, -0.3291725218296051, -1.0392251014709473, -1.1534892320632935, 0.8460588455200195, 0.7076718807220459, -1.8460873365402222, -1.2394167184829712, -0.5707194209098816, -0.8669734597206116, 0.24190694093704224, -0.8737456202507019, 0.17197465896606445, 0.6808804869651794, -0.1413027048110962, -0.22226539254188538, -1.0242388248443604, 0.33089229464530945, 1.390726923942566, -1.314354658126831, 0.19286946952342987, 0.8022393584251404, 1.3586678504943848, 1.1000258922576904, 0.3557104170322418, -0.42528894543647766, -0.8001049757003784, 0.3130147457122803, 0.034491486847400665, -0.18639366328716278, -0.7961927652359009, -2.145097255706787, -1.384517788887024, -0.9597668051719666, -1.1245390176773071, -0.7295556664466858, 0.062008120119571686, -1.2767252922058105, 1.755023717880249, -1.9492971897125244, 0.8832931518554688, 0.5229344367980957, -1.1560250520706177, 2.271343469619751, -0.41381940245628357, -2.263108015060425, -0.4092719554901123, 1.0363459587097168, 0.6402897834777832, 1.8861167430877686, 0.9900254011154175, 0.7586880922317505, -0.8099092245101929, -1.2291158437728882, 1.4626007080078125, -0.1671942174434662, 0.34236788749694824, -1.5164741277694702, 1.096719741821289, -1.0409135818481445, -0.9070073962211609, -0.259469211101532, 0.13410601019859314, -0.5645686984062195, 0.5967978239059448, 1.649841547012329, -2.0602433681488037, 0.3772052824497223, 3.0742545127868652, 0.4764154553413391, -0.09314277768135071, 1.4133163690567017], [0.2346336990594864, 1.9327391386032104, -0.7478625774383545, -0.568943440914154, -1.480857253074646, 0.011847829446196556, -0.29085978865623474, 0.1281331181526184, 1.140389323234558, 1.0788344144821167, -0.2116331309080124, -0.6297260522842407, 1.257180094718933, 0.6303170919418335, 0.5316563844680786, -0.2314174473285675, 1.155576467514038, 0.8541262745857239, 0.8648840188980103, -0.9393776655197144, -0.67002934217453, -0.5324909687042236, 0.18487872183322906, 0.9078885316848755, -0.20864760875701904, -0.9535794854164124, -0.4968765676021576, -0.43620309233665466, -1.0516325235366821, 0.3633269965648651, 0.3009876012802124, 1.8276221752166748, -0.2304142266511917, 0.19045698642730713, -0.377477765083313, 0.12604159116744995, 0.3060070872306824, 0.6923436522483826, 0.5416049957275391, -0.7506083250045776, 0.3996361494064331, -0.4776748716831207, 0.23703499138355255, 0.31869545578956604, 1.1581473350524902, -0.42662590742111206, -0.38900429010391235, 0.6558471322059631, -1.0010502338409424, 0.21525363624095917, -1.3117995262145996, -1.1373859643936157, 0.2575671374797821, 2.2569079399108887, -1.8096977472305298, 0.6256672739982605, 2.8168158531188965, -0.4539409875869751, -1.1102765798568726, -2.2321155071258545, 0.373675674200058, 1.0839893817901611, 1.6458933353424072, -0.019087884575128555, 0.680336594581604, 1.1723183393478394, 0.7102546095848083, 1.261893630027771, -1.0676015615463257, -0.8098644018173218, -0.26293423771858215, -0.652695894241333, 0.7876121401786804, 0.6561001539230347, 1.212000846862793, 0.09169898927211761, 1.0977938175201416, 0.5417161583900452, -0.10138393193483353, 0.01903916709125042, -1.2921826839447021, -1.870725393295288, 0.25729990005493164, 1.4026869535446167, -0.6112109422683716, -1.1128870248794556, 2.175509452819824, 0.45374664664268494, -0.8890997767448425, -0.40592288970947266, -0.6132056713104248, -0.6498266458511353, -1.4312008619308472, 1.824373722076416, 0.9386568069458008, 0.07099085301160812, 0.10298477113246918, -0.6334248781204224, 0.7795170545578003, -0.704034686088562, 0.3436775505542755, 0.06730353087186813, -0.36322638392448425, 0.7649755477905273, -0.6068649291992188, -0.9117646813392639, -0.015538318082690239, 0.01811235584318638, 2.662024736404419, 1.141591191291809, -0.9952392578125, -1.0741957426071167, 0.1857559084892273, 0.33382648229599, 1.2250252962112427, 0.4908706545829773, -0.2688627541065216, 0.6045913696289062, -0.3959903120994568, -0.8269343972206116, 0.9153565764427185, -0.5517545938491821, -0.44276612997055054, 0.823866605758667, -0.8836714029312134, 0.5877604484558105, 1.2623111009597778, -0.935555636882782], [1.585586428642273, -0.35090237855911255, -0.05227169394493103, 0.9055235981941223, 1.0963380336761475, -0.5295907855033875, -0.8424914479255676, -0.37012794613838196, -0.7148021459579468, -0.7213386297225952, 0.5734818577766418, 1.9312673807144165, -0.23583443462848663, -0.38878732919692993, 2.188711166381836, -1.0518052577972412, 0.5183076858520508, -0.38103410601615906, 0.24544332921504974, -0.7007955312728882, 0.2984616160392761, -0.46014097332954407, -1.1131856441497803, -1.2289931774139404, 0.8660207390785217, 2.176515579223633, 0.5105005502700806, -0.29261913895606995, 0.48033639788627625, 1.5159410238265991, 0.15005619823932648, 1.0071687698364258, -0.2795373201370239, -0.9724514484405518, 0.40567126870155334, -1.8396859169006348, -0.997240424156189, 0.034541357308626175, -2.258657932281494, -2.3481578826904297, -0.32729196548461914, -0.8065522313117981, 0.11367925256490707, -0.20907384157180786, -0.6077299118041992, 1.3650140762329102, 1.2310477495193481, 0.07424461096525192, 1.5056238174438477, 0.3114829957485199, 0.6940680146217346, 0.350321888923645, 0.4707205295562744, -0.2690844237804413, -0.2365391105413437, 0.05463311821222305, 0.5492784976959229, 1.886427879333496, 1.391731858253479, 0.005103515461087227, -0.08364060521125793, -1.1342424154281616, -0.849655032157898, -1.188201665878296, 0.02043653093278408, 0.7828440070152283, -1.0439225435256958, 1.5140830278396606, -0.7904909253120422, 0.8054107427597046, 1.5522494316101074, -1.1078599691390991, 2.012098789215088, 0.6539416909217834, 0.9589526057243347, 0.21862515807151794, 1.5456907749176025, 0.26212078332901, 2.1345326900482178, -0.5195248126983643, -0.8828580975532532, 1.1271576881408691, -0.049451593309640884, 2.1416139602661133, -0.8447321057319641, -1.0183138847351074, 0.05976644530892372, -0.21706530451774597, -0.051052406430244446, 0.8273817896842957, -1.0221704244613647, -0.7286556363105774, -1.3005369901657104, 0.3707429766654968, 0.5027804374694824, -1.9336857795715332, -0.5088558197021484, 0.17284274101257324, 1.6375834941864014, 0.863746702671051, 0.44063785672187805, -0.8067634701728821, -0.55415278673172, -0.36855512857437134, 0.676840603351593, 2.031519889831543, -0.9152410626411438, 0.12078574299812317, -0.7571761608123779, 1.1170976161956787, 0.5470037460327148, 0.6679972410202026, -0.8529043793678284, -0.29216766357421875, 0.45746585726737976, 0.7088279128074646, -0.6839810609817505, -0.16144968569278717, 1.6823369264602661, -0.949944019317627, -0.36553823947906494, -0.06502699106931686, 0.5160516500473022, 0.18984058499336243, 0.5436108708381653, 0.812192976474762, -0.9674621820449829, 0.22998277842998505], [1.2328518629074097, -0.7829192280769348, 0.6145099401473999, -0.9402341842651367, -0.9093976020812988, -0.1663333773612976, -1.0441428422927856, -0.058147188276052475, 0.24497883021831512, -1.173237919807434, 0.016397371888160706, 0.6039869785308838, -0.9998157024383545, 0.8380177617073059, 0.6275432109832764, -1.6242172718048096, 1.0020097494125366, -1.2400001287460327, 0.4958690106868744, -1.3648947477340698, 0.1144089624285698, -1.5307798385620117, -1.6166281700134277, 0.21220754086971283, 0.27813369035720825, -1.9928693771362305, 1.0629278421401978, 0.3995806574821472, -0.44964879751205444, 0.13211920857429504, 0.06387435644865036, 0.5214953422546387, -0.3671363294124603, 1.1790708303451538, -0.9955450296401978, 2.3497064113616943, 0.025057006627321243, -0.2650187909603119, 0.6545522212982178, -0.05405480042099953, -0.012846329249441624, 1.1322838068008423, 1.6121848821640015, 1.3144147396087646, 0.1691461205482483, -0.023388994857668877, -0.6833286285400391, 0.7688048481941223, 0.4195597767829895, 0.8545041084289551, 2.063606023788452, -1.0722119808197021, -0.5051915049552917, 0.4213438928127289, -0.3310415744781494, -0.11069173365831375, 1.1754907369613647, -0.6374839544296265, -1.3004276752471924, 0.9418479204177856, -0.4904184937477112, -0.2826816141605377, 0.0646372064948082, -0.8285999894142151, 0.0249467846006155, -1.2207101583480835, 0.5180232524871826, 1.1439720392227173, 1.6313838958740234, 2.7076265811920166, -1.1193578243255615, 0.19237282872200012, 1.7876490354537964, -0.45995432138442993, 0.8993526697158813, -1.2911444902420044, -0.8892064094543457, 1.0668931007385254, 1.6213734149932861, -0.6460960507392883, -1.1351635456085205, -0.9749360680580139, 1.0625144243240356, -0.6868239641189575, -1.7731035947799683, 0.47676944732666016, 0.4237326383590698, -0.5120992660522461, -1.6898640394210815, 2.0874240398406982, 0.2949085533618927, -0.8340200781822205, 1.8991419076919556, 0.3598354756832123, -0.7149474620819092, -0.9580086469650269, 1.4361652135849, -0.9743090867996216, 0.3343872129917145, -1.6558012962341309, 0.13953208923339844, -0.5795975923538208, 0.9184026122093201, 0.811393141746521, -0.7423930168151855, 0.11611910164356232, -1.4644877910614014, 1.4086458683013916, 0.7848925590515137, 0.2932073771953583, 0.9843947887420654, 0.5907356142997742, 0.09233549237251282, 0.1134830117225647, -0.41573652625083923, 3.259340763092041, -0.5208158493041992, -2.2646143436431885, -0.37897494435310364, 0.6413764953613281, -0.14331211149692535, -1.2286158800125122, 0.23104117810726166, -0.9539704918861389, -1.6042842864990234, -0.37544944882392883, 0.8983903527259827, 0.11680705100297928], [-0.21537791192531586, -0.24908855557441711, -0.5288270115852356, 1.2257276773452759, -1.6581885814666748, -0.9743111729621887, -0.0668027400970459, 0.5861430764198303, 0.30099865794181824, -0.5684199929237366, -0.4906638562679291, -2.5178465843200684, 0.018748881295323372, -0.14239534735679626, 0.3183229863643646, -0.0594254732131958, -1.179267168045044, -1.355997085571289, -1.0131723880767822, -0.30771705508232117, 2.8946216106414795, -0.05299459397792816, -2.686392307281494, 0.10462722927331924, 2.4518070220947266, -2.2745749950408936, 1.0108304023742676, -0.5727227926254272, 1.0672168731689453, -0.9213963747024536, 0.5473336577415466, 0.4474506676197052, -0.7877812385559082, -0.5010673999786377, 0.3400498926639557, 0.20703552663326263, 0.14906756579875946, -0.34476161003112793, 0.10789866745471954, -0.05312671884894371, 1.5004535913467407, 1.2397403717041016, -1.8511111736297607, 1.940181016921997, -0.1257268488407135, -0.32941025495529175, -0.673433244228363, -0.22307121753692627, -1.0760761499404907, -0.42852941155433655, 0.6525692939758301, -1.1988670825958252, 0.5788227915763855, -0.997555136680603, -0.4964514970779419, -0.5229364633560181, -0.8150097131729126, 0.22052176296710968, -2.2319886684417725, 1.6264233589172363, -0.09750867635011673, -1.2097355127334595, 0.014646872878074646, -0.8696702122688293, -1.3773561716079712, -0.7215336561203003, -0.17446374893188477, -1.2053486108779907, 0.40886491537094116, -1.07378351688385, -1.080857753753662, -1.2396647930145264, 0.4455191493034363, 0.22254954278469086, 0.40081265568733215, -0.9872320294380188, -0.5286598205566406, -0.34179210662841797, -0.7212871313095093, -0.3281656503677368, -0.1191454529762268, 0.32593533396720886, 1.512208104133606, 0.7940516471862793, -0.3592696189880371, 1.2099480628967285, -1.5514438152313232, 0.8044108748435974, -0.9372342228889465, -2.292888641357422, -0.5562186241149902, 0.4441279470920563, -0.022254828363656998, -0.824822723865509, 0.09060714393854141, 0.0325821153819561, 1.2776751518249512, -0.7786887288093567, -1.9904390573501587, -0.5113791227340698, -0.012715181335806847, -0.8703904747962952, -0.23314924538135529, -0.47022974491119385, 0.3645261228084564, -0.5601003170013428, -0.7819127440452576, 0.38213780522346497, -0.7457538843154907, 1.1922075748443604, -0.015130911953747272, 0.6313763856887817, -0.8843415975570679, -0.1668294221162796, 0.5693690180778503, -0.6424914002418518, -0.4118642807006836, -0.10656608641147614, 0.8989484310150146, 0.2918268144130707, 0.2123689502477646, -0.32903367280960083, 1.4942002296447754, 1.0211586952209473, 0.8989136219024658, -0.39458832144737244, -0.6794130802154541, -0.5565900802612305], [-0.674240231513977, -1.0132516622543335, 1.3773596286773682, 0.3214305341243744, 1.1470907926559448, -0.5635467171669006, 1.2863346338272095, 0.9644817113876343, -1.1432746648788452, 0.12738487124443054, 0.07356376945972443, 1.82882821559906, -2.4335551261901855, -1.0036579370498657, 1.434424638748169, -2.195613145828247, 0.6519201993942261, -1.3003889322280884, -0.08330120146274567, 0.5506670475006104, -0.31121477484703064, 1.068719744682312, -0.24836860597133636, 0.9289913773536682, 1.8380966186523438, 0.2727498412132263, 0.5303915739059448, 0.9247944355010986, -1.4066400527954102, -1.5304739475250244, 0.738508403301239, 1.4150409698486328, -0.08765190094709396, -0.3919057846069336, -0.702613890171051, -0.8816413879394531, -0.8833340406417847, 2.383774518966675, 1.5275846719741821, 1.4257864952087402, -0.4043821692466736, 0.1232648715376854, 1.3586117029190063, 2.0384931564331055, 0.5675883889198303, -0.7002530694007874, 0.2938421666622162, 0.059850938618183136, 1.9099467992782593, 1.2082538604736328, -0.6582364439964294, 0.34674128890037537, 1.340933918952942, -1.3577873706817627, -1.7773535251617432, 0.44317787885665894, -0.3518420159816742, -0.23654450476169586, -1.1536070108413696, -1.782918930053711, -0.7730772495269775, -0.2178792655467987, -0.8659269213676453, -0.21606627106666565, -1.0952510833740234, 1.0456191301345825, 0.07281282544136047, 0.35079649090766907, 0.6296545267105103, 0.8821863532066345, 0.8136013746261597, 0.0781627893447876, -0.38341131806373596, 0.6880490183830261, -1.1630607843399048, 1.20192289352417, 0.4006898105144501, 0.29809704422950745, 0.3227187991142273, 0.6387341022491455, 0.8710342049598694, 0.2147177755832672, -0.14911910891532898, 0.35088104009628296, -1.5215173959732056, -0.44979169964790344, 0.18100093305110931, 0.732603907585144, 0.7068331837654114, -0.47808972001075745, -0.44050902128219604, 0.32215702533721924, -0.05359762907028198, 0.4740714430809021, 1.8658568859100342, 1.0234483480453491, 0.3999754786491394, 0.8357415795326233, 0.5088079571723938, -1.279765248298645, 1.411785364151001, -0.5089505910873413, 0.07592131942510605, 1.6357166767120361, 1.0210610628128052, 0.2729234993457794, -1.0938607454299927, 1.397904396057129, 1.1362016201019287, -0.4976409673690796, 1.2427626848220825, -0.37409260869026184, -0.4607597589492798, 1.0808969736099243, -0.782856822013855, 0.8716723322868347, 0.16421882808208466, -1.5775182247161865, 0.48496851325035095, -0.8720828890800476, 0.7563187479972839, -1.1952658891677856, -0.2818911373615265, -0.2660232186317444, 2.126814842224121, -1.0510108470916748, -0.2794557213783264, 1.3939481973648071], [0.4064972400665283, 1.0650107860565186, 0.3692023456096649, 1.4205464124679565, -0.16379070281982422, 0.1115143820643425, -1.8203299045562744, 0.3428545892238617, 0.537078857421875, 1.2404872179031372, -0.590226411819458, 1.618489146232605, -1.4768621921539307, -1.1909009218215942, 1.0111278295516968, 2.3244850635528564, 0.7195126414299011, -0.6291541457176208, -1.0688797235488892, 0.3571288287639618, -0.13361135125160217, -0.932437002658844, 0.10620126873254776, 1.129898190498352, 2.503598928451538, -0.9807766675949097, -0.2348024994134903, -0.1970888376235962, -0.2978806793689728, -0.9906725883483887, 0.09026689827442169, 1.3105087280273438, 0.1661110520362854, 0.26711076498031616, 0.9940507411956787, 1.2549811601638794, -0.2629713714122772, 1.9109970331192017, 1.3449536561965942, 1.915321707725525, 1.1915390491485596, -0.4256631135940552, -1.3258485794067383, 2.3538315296173096, -0.3095678687095642, -0.2197503000497818, -1.2816038131713867, 0.3012407422065735, 0.5460547804832458, 0.3689385652542114, -0.322925329208374, 0.09962044656276703, -0.34268414974212646, 0.7492766380310059, -0.23745019733905792, -1.2968169450759888, 0.5966785550117493, -0.24573799967765808, -1.7880905866622925, -1.4660347700119019, -0.989502489566803, -1.3133127689361572, -0.850055992603302, -0.7020485997200012, 0.016325082629919052, -1.1690057516098022, -1.0602318048477173, 0.9259688258171082, 1.648675799369812, 0.5152929425239563, 0.9167160391807556, -1.8847631216049194, -0.3195393979549408, 0.1827744096517563, 1.0095155239105225, 0.9443264007568359, -0.8213995695114136, 1.0474179983139038, -0.6912565231323242, -0.597322940826416, 0.2257184535264969, -0.8344758749008179, -0.007550508249551058, -0.1574755758047104, -1.0412839651107788, 0.6528223156929016, -0.7884113788604736, 1.287585973739624, 0.21311762928962708, 2.6432459354400635, -1.4570099115371704, -1.5598610639572144, -0.05972305312752724, -0.695846676826477, -1.011271595954895, -0.6357024908065796, 0.023808740079402924, -0.024375829845666885, 0.04127219319343567, 1.5205636024475098, 0.5127003192901611, -1.4300165176391602, 0.9752975702285767, 0.22795642912387848, -0.5184799432754517, -0.6237280368804932, -0.8222517967224121, -0.5707696080207825, 1.8836729526519775, -0.4239179491996765, -0.3013969957828522, 0.03454264625906944, 1.5871862173080444, 2.53175687789917, 0.6238283514976501, -0.9414236545562744, -0.21650950610637665, -0.6464626789093018, -0.2173851728439331, 0.09243492037057877, 0.39489489793777466, 0.8882998824119568, -1.576674461364746, 1.643601655960083, -1.1809911727905273, -0.0006683069514110684, 0.9196714162826538, 1.1069419384002686], [-1.0367158651351929, 0.23534540832042694, -0.4707544147968292, -0.7344451546669006, 0.9733754396438599, 2.0529568195343018, 0.7940793633460999, 0.13777141273021698, 0.7904369235038757, -1.2150413990020752, 0.38816773891448975, -1.2180440425872803, 1.6376887559890747, -0.9327693581581116, 0.3434731364250183, 1.149548053741455, 0.1271767020225525, -0.7022087574005127, -0.03760038688778877, -0.7043320536613464, -0.6888239979743958, -1.3527979850769043, -0.6697682738304138, 0.6248807907104492, -0.7660486102104187, 0.101980060338974, -0.6748239994049072, -0.7274016737937927, 0.05581360682845116, -0.27062562108039856, -1.1373915672302246, -0.5469127297401428, -0.3517482280731201, -0.5415732860565186, -0.4434482753276825, -1.0414923429489136, 0.1621556431055069, 0.23425038158893585, 1.288338541984558, -1.1451590061187744, -0.276770681142807, -0.7738783359527588, -1.1729782819747925, 0.5173197984695435, -1.9139066934585571, -0.06168169528245926, 0.7853729724884033, 0.8895204663276672, -0.1110558956861496, -0.9756402969360352, 1.0853958129882812, -2.011212110519409, -0.21926145255565643, -0.8550115823745728, -1.56882905960083, 0.5873135328292847, 1.0819330215454102, 0.6041972637176514, 0.7173968553543091, -0.5560500621795654, 1.4884790182113647, -0.9170870184898376, 1.9360002279281616, -0.5104759335517883, 0.6262492537498474, -0.8557541370391846, -1.1197091341018677, 1.4796913862228394, 0.8513739705085754, -1.2482695579528809, -0.8662524819374084, 0.3583441376686096, -1.3271969556808472, -0.3022399842739105, -0.6537869572639465, -3.0941877365112305, -0.2992187440395355, 0.21759232878684998, 0.8569901585578918, -0.7570238709449768, 0.6967688798904419, 0.5315525531768799, -0.2818375527858734, -2.1168372631073, 0.6231086254119873, 0.6019924283027649, 1.1323729753494263, -0.37843403220176697, 0.034506458789110184, -0.4920010566711426, 0.5198871493339539, 0.8917905688285828, -1.2399506568908691, 0.02494954690337181, -1.2498401403427124, -1.9315195083618164, -0.607073187828064, 0.5484729409217834, -1.3981683254241943, 0.6506755352020264, 0.3073924481868744, -0.3888128697872162, 1.890377163887024, 0.7949628233909607, -0.2095613181591034, 0.2025107741355896, 1.1263389587402344, 0.23423363268375397, 0.05819429084658623, 0.42811280488967896, 0.9635310173034668, 0.20764988660812378, 0.9951127171516418, -1.0387802124023438, 0.770408570766449, 1.3419196605682373, 0.20675712823867798, -0.33616864681243896, 1.4151878356933594, 1.2380620241165161, -1.0207464694976807, -2.124865770339966, -0.9548389911651611, -0.1481345146894455, -0.06210608035326004, -0.7708704471588135, 0.1355820745229721, 1.3567891120910645], [-1.3289790153503418, -0.5736090540885925, 1.174160361289978, 1.186328649520874, -0.32595863938331604, -0.7298804521560669, 0.12167558819055557, 1.3301177024841309, -1.1832376718521118, -0.056204285472631454, 1.0598576068878174, 0.5492715239524841, -1.1610760688781738, -1.4986132383346558, -0.9224693775177002, -2.060547351837158, 0.9428311586380005, 0.7329380512237549, 0.531619668006897, 0.027045994997024536, -0.09203799068927765, -1.5837918519973755, 0.08621682226657867, -1.329519510269165, -1.2113689184188843, -0.8123772144317627, -0.8091709017753601, 0.21276433765888214, 0.42707201838493347, 0.3666478097438812, -0.8774981498718262, -0.027498023584485054, -1.2172456979751587, -0.5618786215782166, 0.9673067927360535, 1.805688500404358, 0.891081690788269, 0.08739425241947174, 0.0661264955997467, -0.5204229950904846, 0.561225414276123, -1.798298716545105, -0.5165286660194397, -0.033067379146814346, 1.1876527070999146, 1.5278266668319702, -1.6177785396575928, 0.5576158165931702, -1.3685944080352783, 0.6647096276283264, 1.1386878490447998, -0.10693083703517914, 0.9142453074455261, -1.2168912887573242, 1.3218094110488892, -0.9225838780403137, 1.0931733846664429, -0.1468620002269745, -0.23457179963588715, 1.223257303237915, 2.0170750617980957, -0.3258334994316101, 0.6568962335586548, 0.29868462681770325, 0.7931969165802002, -0.659874439239502, 0.34920671582221985, -0.1476467251777649, -0.6590886116027832, -0.9898090362548828, -1.4030755758285522, -1.680986762046814, 0.3930933177471161, 0.17508111894130707, -0.31630632281303406, -1.1937578916549683, -0.8331581950187683, -1.8689988851547241, 0.20060038566589355, -0.25838756561279297, -1.8650027513504028, 1.5683873891830444, -1.7321921586990356, 0.6529525518417358, 1.349269151687622, 0.7381157279014587, -0.13602399826049805, 1.9783878326416016, -0.5599006414413452, 0.4952446222305298, -1.2251043319702148, -0.7056388854980469, 0.6308253407478333, 1.3728965520858765, 0.6663592457771301, -0.6689398884773254, -1.6983799934387207, 0.28480958938598633, 1.2992147207260132, 0.40382665395736694, -0.9129557013511658, 0.6321684718132019, 0.8575639128684998, -0.8938085436820984, 0.39333662390708923, -0.9802290201187134, -0.13743065297603607, -1.3707793951034546, -0.4152974486351013, -1.7717620134353638, 0.6741936802864075, 1.2790335416793823, 0.834955632686615, -1.3599187135696411, -0.9479265809059143, 0.3250482380390167, -1.28084135055542, -0.9652577042579651, -0.9080570340156555, -0.651589572429657, 0.8062693476676941, 0.773975133895874, -0.3687427341938019, 1.7786905765533447, 0.6609163880348206, -0.33192178606987, -0.7342920899391174, -1.4933116436004639], [1.4654463529586792, 0.18367524445056915, 0.8454532027244568, 2.692744255065918, 1.69643235206604, -0.4285966753959656, 1.8963803052902222, -0.8167949318885803, 1.894504427909851, 0.00656706839799881, 1.2212823629379272, -1.0064138174057007, -0.4883613586425781, -0.3419985771179199, 0.17961695790290833, 0.672812283039093, -0.8844308853149414, 0.2541934847831726, -0.9553385376930237, -1.133553147315979, -0.918117880821228, -0.6863341331481934, -0.9465499520301819, 0.49146726727485657, 2.1672704219818115, 0.36623191833496094, -2.206770896911621, -0.17009437084197998, -0.5782415270805359, -0.22912965714931488, 0.2528241276741028, 1.4947127103805542, 2.2256290912628174, 0.1219683289527893, 0.34304317831993103, 0.6220210790634155, -2.387253761291504, -0.6350299715995789, 0.9450433850288391, -0.3573034703731537, -0.7809363007545471, -0.13732001185417175, -0.5331906676292419, -0.447179913520813, -2.065229654312134, -0.9879633784294128, 1.029809594154358, -1.4723832607269287, -0.08167935162782669, -1.0394971370697021, -0.7856453061103821, 0.25957995653152466, 0.7472455501556396, 0.5257085561752319, 0.5490821003913879, -0.5101768970489502, 0.9809291362762451, 0.6125028729438782, 0.06408052891492844, 0.9711549878120422, -0.36528968811035156, 0.46116751432418823, -0.085933156311512, -1.4882011413574219, 0.7986642718315125, 1.1550800800323486, 0.04486178234219551, 0.355960488319397, -0.2688848078250885, 2.0447959899902344, -1.1426640748977661, -1.5772212743759155, 0.3224051296710968, 0.5033411383628845, 0.4650344252586365, 0.530755877494812, -1.8971534967422485, 0.21797938644886017, 1.0561870336532593, -0.5933972597122192, -0.15789996087551117, -0.285077840089798, -0.684622049331665, -1.762044072151184, -0.07674819231033325, 0.3826010823249817, 2.528510808944702, 0.95737624168396, -0.4038330316543579, 0.8155797719955444, 0.9876224398612976, -1.481059193611145, -0.47185370326042175, -0.1894838511943817, 1.0252394676208496, -2.3346595764160156, 1.7707273960113525, 0.6100795865058899, -0.4055287539958954, 0.4271656572818756, 0.5324392318725586, 1.5585534572601318, 0.1042928472161293, 0.5593857169151306, -1.3852373361587524, 0.3681504726409912, -0.740074872970581, 0.286833792924881, -0.5528731942176819, 0.9198652505874634, 0.30720582604408264, -0.19045883417129517, 1.0665695667266846, -1.1712480783462524, 0.2258911430835724, -0.9796176552772522, 0.9155861139297485, -0.7171744108200073, -0.8691605925559998, 0.3268047869205475, -1.3392236232757568, -0.7591859102249146, -0.09915575385093689, -0.5963033437728882, -0.10574761033058167, 0.23859049379825592, -1.5766488313674927, -0.6275414824485779], [-1.3353960514068604, -2.9953420162200928, -0.752410888671875, -0.0819159597158432, 0.07523718476295471, 0.4901566803455353, 0.5501279830932617, 0.31843259930610657, 0.4041394591331482, 2.1168978214263916, 0.006760165095329285, 1.2318638563156128, 0.24338765442371368, -0.2001960277557373, -0.9790597558021545, 0.037579890340566635, 1.8888429403305054, 0.35836851596832275, -1.5198793411254883, 1.4385184049606323, -0.2525641620159149, -0.5768795609474182, 0.7443480491638184, -1.46532142162323, -2.0010523796081543, -0.3041386008262634, -0.11115840822458267, 0.964914083480835, 0.0032116721849888563, -0.025430472567677498, -1.069064974784851, 1.8532435894012451, -0.9465927481651306, -0.7568612098693848, 1.7082171440124512, -0.7278295755386353, 0.7506469488143921, 0.11896952986717224, -0.33989009261131287, 0.41469499468803406, 0.6610864996910095, -0.9810652136802673, -0.0029577072709798813, 0.98923659324646, 2.1963634490966797, -0.1542646437883377, 0.8417974710464478, -0.09170790761709213, 1.4681848287582397, 0.8846219778060913, 0.6617227792739868, -0.8361130356788635, 0.4143471121788025, -0.8546179533004761, 0.3102075159549713, -1.3162455558776855, -0.6531413197517395, -0.2689206302165985, -0.9466173052787781, 1.207730770111084, 0.42087382078170776, 0.09948636591434479, -0.7906738519668579, 0.8303229212760925, 0.6569917798042297, -0.3574390411376953, -0.2888610363006592, 0.5108484625816345, -0.4255475401878357, -1.7094727754592896, 1.8202592134475708, -0.7346004247665405, 0.27524274587631226, -0.3629821240901947, 0.19616153836250305, -0.939215898513794, 0.0565144382417202, -2.1108741760253906, -2.743154525756836, -1.5262773036956787, 0.27592265605926514, -0.21024636924266815, 1.0114870071411133, -0.4983111023902893, -0.5454820394515991, -0.8924893140792847, 1.8766202926635742, 0.38884714245796204, 0.07055965065956116, -0.8702963590621948, 1.5767033100128174, -0.061063386499881744, 0.28146377205848694, -0.5529982447624207, -0.44201165437698364, 0.5689774751663208, 0.1716163158416748, -0.9487428069114685, -0.6204344034194946, 0.5324710011482239, 1.5306649208068848, 0.002167449565604329, 0.2944130599498749, 0.30758923292160034, 0.4180241525173187, 0.34772709012031555, 0.9384362101554871, -2.4354593753814697, 0.32328447699546814, -0.7376670837402344, -0.9894570112228394, -0.3836098611354828, -0.7627635598182678, -0.24811096489429474, -0.33239057660102844, -0.8255242109298706, 0.9021656513214111, 0.46810874342918396, 0.7118749022483826, 2.511974573135376, -2.1563467979431152, 0.04314689710736275, 1.325605869293213, 0.533221960067749, 1.2628639936447144, 1.3798589706420898, 0.3680863678455353, 0.5794890522956848], [0.2405954748392105, -1.5118497610092163, 0.325756698846817, -0.6411396265029907, -0.10878387838602066, -1.6862951517105103, 0.0105389803647995, -0.1775815337896347, 0.609831690788269, -2.4360618591308594, 1.3537592887878418, -0.3525693118572235, 0.03195798769593239, 0.15705552697181702, 0.7501788139343262, -0.6640225052833557, 1.6560847759246826, 2.968665361404419, -0.6970561742782593, -0.645016610622406, 0.8968585133552551, -0.684639573097229, -0.7738004326820374, -1.5783660411834717, 0.1425619125366211, -1.0230108499526978, -0.5610824823379517, -0.5795208811759949, 0.5432005524635315, 2.01657772064209, 0.154083251953125, -0.5456662774085999, 0.4899080693721771, -0.6834473609924316, 0.4596123993396759, -0.6987388730049133, 2.3748607635498047, -2.0868797302246094, 0.3702852129936218, 0.3620935082435608, 0.2951459288597107, -1.6777548789978027, -0.9965100884437561, -0.8253192901611328, -0.15012574195861816, 0.6689980626106262, -1.1369463205337524, -1.324905514717102, 1.876522421836853, 0.10656296461820602, -0.3187013566493988, 0.17787747085094452, 0.4116925001144409, 1.0942537784576416, 0.41774922609329224, 0.4567722678184509, 0.8565411567687988, -0.22038163244724274, 1.3259940147399902, -0.10706070810556412, -0.41475576162338257, 0.9516972899436951, -1.0755054950714111, -0.6101139187812805, 0.13174740970134735, -0.8949400782585144, -0.8975166082382202, -0.20642732083797455, -0.8277410268783569, 0.09797857701778412, -0.2024964988231659, -0.43800801038742065, -0.7481237649917603, -0.8692882657051086, 0.6710245013237, 2.341571807861328, -0.16174612939357758, 0.564007580280304, -0.5786256194114685, -0.724014163017273, -0.02925647422671318, 0.8982763290405273, 1.4901623725891113, -1.5668164491653442, -0.16988743841648102, 0.5762169361114502, -0.07742789387702942, -0.3744259774684906, 1.275352954864502, -0.09338173270225525, 0.4521394968032837, -0.10644448548555374, -0.6073655486106873, 0.46388566493988037, 0.33463549613952637, -1.0587915182113647, 1.0036908388137817, -0.5835511684417725, 1.955021858215332, -0.2235252410173416, 0.968438982963562, 0.46856334805488586, 0.059542205184698105, 0.6568491458892822, -0.8561244010925293, 1.496412754058838, 1.5806496143341064, -2.353105306625366, -2.827162265777588, 1.025349497795105, 0.7108051776885986, -0.049762677401304245, 2.2177534103393555, -1.6787431240081787, -0.05030355975031853, -1.7309929132461548, 2.184894561767578, -1.9461942911148071, -0.8448305130004883, 0.5806670784950256, 1.187282919883728, -1.197643756866455, -0.5133927464485168, -0.3735547959804535, 0.6530118584632874, 0.6772355437278748, -1.9866960048675537, 1.1233936548233032], [1.2601014375686646, 1.8223638534545898, 0.2562079429626465, -0.3913002014160156, -0.20323620736598969, 0.776738703250885, -1.9361907243728638, 1.4589247703552246, 0.8147551417350769, 0.6487510204315186, 1.1064056158065796, -1.2532484531402588, -0.04064689576625824, -0.33987727761268616, -0.8474823236465454, -1.1528809070587158, 0.6809943318367004, 0.7126211524009705, -0.15538668632507324, -0.5600141286849976, 0.19111983478069305, 0.676646888256073, 0.08239098638296127, 0.18178944289684296, -0.724176824092865, -1.4165847301483154, 1.0906658172607422, 1.8643234968185425, 0.49306100606918335, -1.2126277685165405, -0.12988600134849548, 0.46215641498565674, -0.1398874968290329, -0.9523211717605591, -1.018816590309143, -0.22794583439826965, -0.17369060218334198, 0.08452930301427841, 0.8311104774475098, 0.6768745183944702, -1.572495460510254, 1.7346739768981934, 1.82180917263031, -0.12408078461885452, -1.0052731037139893, 0.7862291932106018, -2.229994297027588, -0.06793899089097977, 1.628556489944458, 0.8149476647377014, 0.1921597123146057, 0.5594764947891235, 1.8144402503967285, -0.9541102647781372, 1.2850528955459595, -1.419007420539856, -0.28673216700553894, 0.285288542509079, 0.12314010411500931, 1.6574727296829224, -1.0228745937347412, 0.02359776198863983, 1.129551649093628, 1.165181279182434, 1.765504002571106, -0.6931970119476318, 1.2034683227539062, -0.1141085997223854, -1.9150561094284058, -1.6384563446044922, 0.4004691243171692, 0.19281291961669922, -0.6793351173400879, 0.805971622467041, 0.1429070085287094, -0.6267737746238708, 0.48196345567703247, -1.4448561668395996, 1.0832126140594482, -1.2013691663742065, 0.34685468673706055, 0.9057902693748474, 1.4961774349212646, -0.6613532304763794, 0.4367489218711853, -1.3465543985366821, 0.759250819683075, 0.8888595104217529, 1.5120595693588257, 0.38974347710609436, -1.2724508047103882, 0.42324623465538025, -0.17717064917087555, 0.11651524156332016, 0.10605309158563614, -0.6125962138175964, 0.26377347111701965, -1.486954689025879, -0.7471980452537537, 1.5599009990692139, 1.3213746547698975, -0.5411259531974792, 0.5058639049530029, -0.058367159217596054, 2.8709118366241455, 0.3255251348018646, 1.313984990119934, 1.84244704246521, 0.1084584966301918, -1.6711665391921997, 0.342079222202301, 0.4887911379337311, 0.11322234570980072, -0.7856013178825378, -1.3769004344940186, 0.9114867448806763, -0.3411879539489746, 0.964645266532898, -0.6912674307823181, 0.8222795128822327, -1.4368150234222412, -0.2122395783662796, -0.611889660358429, -0.3051645755767822, -1.1591752767562866, 0.38218507170677185, 0.01795314811170101, 0.7895880937576294], [-1.6072345972061157, 0.5194364190101624, 0.5537908673286438, 0.8029424548149109, 1.2779549360275269, 0.1400994509458542, -0.683135986328125, -0.9759028553962708, 0.46295905113220215, 0.9401808381080627, -0.2178429663181305, 0.5071051716804504, 1.3095303773880005, -0.44828298687934875, -0.89244544506073, 0.3095513880252838, 1.5506740808486938, 0.3326164186000824, 0.5468958020210266, 0.7823431491851807, -1.1548070907592773, 1.376136064529419, -0.7178117632865906, 0.18904675543308258, 0.3140007257461548, 0.7998068928718567, 0.3227768838405609, -0.0007410713587887585, -0.01760825328528881, -1.5842700004577637, -1.6971246004104614, 0.5018733143806458, -0.2441239356994629, 0.19981920719146729, -0.8962860703468323, -0.20050637423992157, 1.3717247247695923, -1.4216196537017822, 0.19110317528247833, 2.2044451236724854, -0.6459079384803772, 0.6141767501831055, 1.357056736946106, -1.5717624425888062, -0.8437735438346863, 0.4911024868488312, 1.0125254392623901, -0.0941896066069603, -1.688604474067688, 1.0138486623764038, 0.16288435459136963, -0.7720513343811035, -1.1141985654830933, -0.2581906318664551, 0.19102834165096283, 0.13718536496162415, 0.421774297952652, 0.7774767279624939, -0.35685983300209045, 0.9950220584869385, 0.9342178702354431, -1.471872091293335, -0.12634143233299255, 0.38104474544525146, 0.010019156150519848, 1.9664301872253418, -1.0671632289886475, 1.0655972957611084, 0.46953076124191284, 0.010890057310461998, 0.4989090859889984, -0.6479557156562805, -0.6327989101409912, 2.9272189140319824, 1.6846016645431519, 1.0241422653198242, -0.8006935119628906, -0.26242774724960327, 0.6417711973190308, 0.03144882619380951, 1.5280739068984985, 0.052839457988739014, 1.208559274673462, -0.37223461270332336, -0.29684433341026306, 0.5447730422019958, -1.3638007640838623, -0.7790406942367554, 0.1923093944787979, 0.5526917576789856, -1.4234334230422974, -0.14175449311733246, 0.181832417845726, 1.5534718036651611, -2.212695360183716, -0.6373133659362793, -1.069146752357483, -0.9346048831939697, 0.4838557541370392, -1.0837326049804688, 0.5817934274673462, -1.2727290391921997, -0.05257035046815872, 0.2171773910522461, -0.0844622477889061, -0.1553105115890503, 1.2619848251342773, -0.0829009860754013, 0.7850319147109985, 0.4691321849822998, 1.5092178583145142, -0.3184289038181305, 0.2541618049144745, -1.691448450088501, -1.4176181554794312, 1.6593139171600342, 0.734061598777771, 1.1367460489273071, 1.7614470720291138, 0.10772277414798737, -0.8937868475914001, -0.05857250466942787, 1.2085503339767456, -0.36445409059524536, -1.1649843454360962, 3.074242115020752, -0.44405391812324524, -0.8293977975845337], [0.8843516111373901, -2.1013479232788086, -0.2879932224750519, 0.5599029660224915, -1.6337608098983765, -0.9368487596511841, 1.5702441930770874, 0.49524179100990295, 0.9806049466133118, 0.7161359190940857, 0.3294510841369629, -0.5811880826950073, -1.594236969947815, -1.382390022277832, -0.20061127841472626, 1.334457278251648, 0.6783052086830139, -0.11445100605487823, -0.6002176403999329, 1.3035310506820679, -0.2606273293495178, -0.7540002465248108, 1.0963199138641357, 1.4770439863204956, -0.22784683108329773, 0.8504202961921692, 0.6627345681190491, -0.517061710357666, -0.13792501389980316, -0.8487701416015625, 0.30604252219200134, 0.058706142008304596, -0.4714978039264679, 0.0344148725271225, -0.21799053251743317, 0.36506715416908264, 0.3043551743030548, -0.11030465364456177, -0.7550432085990906, -0.31558525562286377, -0.077785424888134, 0.04611353576183319, 0.8436932563781738, -0.0906948372721672, 0.06449040025472641, 0.9815415740013123, -0.6223947405815125, -0.9874258041381836, -1.0121262073516846, -0.1288614571094513, 1.2288434505462646, 0.3170555830001831, -0.23856452107429504, -1.1529910564422607, 0.9272667765617371, -1.0922961235046387, -1.540872573852539, 0.015635397285223007, 0.6754646301269531, 0.996623694896698, 1.994925618171692, 0.14809119701385498, 0.14463357627391815, 1.2346984148025513, 1.8172353506088257, -1.3649486303329468, 1.0236759185791016, -0.6807161569595337, 0.8599303364753723, -0.4301096796989441, -0.6050887107849121, 0.6594591736793518, -0.678449273109436, 0.8059970140457153, -0.15292169153690338, -0.4146842360496521, 2.143470287322998, -0.5398251414299011, 0.26549285650253296, 0.9214686751365662, -0.4561522603034973, 0.6259365677833557, -1.0525317192077637, -1.4979139566421509, 1.252479910850525, 0.35200950503349304, -1.63929283618927, 0.2079191952943802, 0.35166990756988525, -0.9098017811775208, 0.45787179470062256, -0.9391568303108215, 0.5440935492515564, 0.8419867157936096, 0.5464408993721008, 1.2363758087158203, 0.7171921133995056, -1.8898717164993286, -0.24350957572460175, 0.534080445766449, 0.894750714302063, 1.5448821783065796, -0.3245275318622589, -1.2992274761199951, 0.17356158792972565, 0.365898460149765, 0.22123682498931885, 0.708713173866272, 1.2595173120498657, 0.8734346628189087, 0.028176987543702126, 0.3324447572231293, -1.3613520860671997, 0.05688006430864334, -0.5586256980895996, -0.04733526334166527, -0.7238932847976685, 1.8550379276275635, 0.47784602642059326, 0.8814315795898438, 0.342276930809021, -1.5845412015914917, 0.2215813845396042, -1.3617655038833618, 1.0378937721252441, 1.5778379440307617, 1.001253366470337, 1.0599048137664795], [-0.40004050731658936, -0.863229513168335, 0.39488527178764343, -0.24166662991046906, 0.761246383190155, -1.1028085947036743, -0.5644538998603821, 1.234241008758545, 0.926334798336029, 2.3809468746185303, 0.9145791530609131, 0.7330376505851746, -1.5776656866073608, -2.2798359394073486, -0.5753487944602966, 0.27334555983543396, -0.3870674967765808, 0.9882023334503174, 0.7665499448776245, -0.20255263149738312, 0.9844247102737427, -0.016052741557359695, 1.1725380420684814, 2.3426475524902344, -1.121146559715271, -1.2348675727844238, 0.6734921932220459, 1.2670308351516724, -1.0026674270629883, 0.6582629084587097, -2.3244404792785645, -0.25422418117523193, 0.36722368001937866, -0.996594250202179, 2.1050963401794434, -1.2889018058776855, 0.2405441254377365, 1.703156590461731, -0.20777647197246552, 0.08536543697118759, 0.39448273181915283, 0.8165550231933594, 1.754411220550537, 0.07916560024023056, 0.7605615258216858, -0.7188020944595337, 0.14472763240337372, -1.4755895137786865, 1.9322541952133179, 0.0815141424536705, 0.31476762890815735, 0.287673681974411, -1.0828356742858887, 0.3762109875679016, -0.23979608714580536, 0.1979803591966629, 0.1580316126346588, 1.2768453359603882, 0.43344366550445557, 0.7600795030593872, -0.6936063170433044, 0.07242348790168762, -1.1496485471725464, -0.37567779421806335, 0.7899165749549866, 0.6661505103111267, 1.3910328149795532, 1.2685257196426392, 0.3003539443016052, -0.0645843893289566, -0.28641584515571594, -0.25580835342407227, 0.16384634375572205, 0.5123049020767212, 0.6911295652389526, 0.17132288217544556, 0.8093776106834412, 0.38928768038749695, 1.1139723062515259, 0.45129549503326416, -0.034898124635219574, 0.9148503541946411, -0.07190041989088058, 0.4305773377418518, 0.9349765181541443, 0.3690364360809326, -1.9559454917907715, -1.5109773874282837, 1.2501561641693115, -0.08094135671854019, -0.325648695230484, -0.4500417113304138, 1.9858030080795288, -1.3953999280929565, 0.31356504559516907, -1.193109154701233, 0.406823992729187, 0.4217297434806824, -0.1734754592180252, 1.065649151802063, 0.24116966128349304, 1.6431571245193481, 0.9437465667724609, -0.9143840074539185, 1.2504119873046875, 0.3707866966724396, -0.6578020453453064, 1.139478325843811, 0.6699252724647522, -0.12566444277763367, -0.30914250016212463, 0.2582826614379883, -0.6803615093231201, -0.3965763747692108, -0.5237388610839844, 0.7511335611343384, 0.4611302614212036, 0.8659946322441101, 1.798223853111267, -1.261795163154602, -0.00011484757851576433, 0.9644383192062378, -0.3043387234210968, -0.9906808137893677, -1.5164496898651123, -0.163413405418396, 0.13487347960472107, -0.5044279098510742], [-1.8153079748153687, 0.8078253269195557, 0.5042873620986938, -0.18223311007022858, 0.020247608423233032, 0.871911883354187, -0.3513241410255432, -0.8225369453430176, -0.7848084568977356, 0.4610625207424164, -0.5424785017967224, -1.9676837921142578, 1.4580405950546265, -0.8914045095443726, 0.7405368685722351, 0.9315861463546753, -1.5186885595321655, 0.7883533239364624, 0.15325354039669037, -0.7948853373527527, -0.7430351376533508, 1.6224263906478882, 0.5352520942687988, -0.2541077733039856, -0.23349271714687347, 1.7094494104385376, 0.31942394375801086, 0.39200636744499207, -0.44271528720855713, -1.2294706106185913, 0.28185251355171204, 0.3730887472629547, 1.448284387588501, -0.7055396437644958, -0.27088218927383423, 0.2488177865743637, 0.18410912156105042, 0.7622668743133545, -1.0913310050964355, 0.6586760878562927, -0.4811401963233948, 0.1368955820798874, 0.26518145203590393, 1.9503062963485718, 0.5960400700569153, -0.44315072894096375, -1.2007495164871216, -1.4966899156570435, 2.1061575412750244, 0.6286896467208862, 1.435515284538269, -0.9331986308097839, -0.3919677734375, 0.334842711687088, 1.6233441829681396, -0.4845096468925476, 1.9705249071121216, 1.0712677240371704, 2.519645929336548, 0.7990158200263977, -0.7030502557754517, -0.1566765457391739, -0.21105556190013885, 0.21314996480941772, -0.28870689868927, -0.7750096917152405, 0.9538158178329468, 1.731351375579834, 0.8320430517196655, 0.39059221744537354, -1.736331820487976, -0.21939148008823395, -0.3801901638507843, 0.22024866938591003, 1.8887757062911987, -1.4509031772613525, 0.6460962891578674, -2.0352518558502197, 0.2227860689163208, 0.46637749671936035, -1.143296480178833, -1.1826304197311401, -0.2191655933856964, 0.972396731376648, -2.776181697845459, 0.5934252142906189, 0.3186947703361511, 0.4020848870277405, 0.9401291012763977, -0.07201085984706879, 0.19159910082817078, 0.12888310849666595, -1.1774555444717407, -2.1630754470825195, -0.04546708986163139, -0.17641650140285492, 0.0893598347902298, -0.5497192144393921, -1.525498628616333, 0.8553206920623779, -0.5559133291244507, 1.7372499704360962, 0.7448265552520752, -2.033073902130127, -0.8202682733535767, -2.078206777572632, -2.3358094692230225, 1.7166342735290527, -0.7527779340744019, 1.6576392650604248, -0.9552232623100281, -1.7253926992416382, 0.2800029516220093, -0.3093585669994354, 0.8741387724876404, -0.6370039582252502, 0.07602062076330185, -1.3445934057235718, 0.04656523838639259, 0.21174581348896027, -0.6251677870750427, -1.2080297470092773, 0.029610373079776764, -1.7022426128387451, -1.2365778684616089, 0.4897584319114685, 1.1075935363769531, 0.36523863673210144], [-0.10857941955327988, -0.5720294713973999, -0.033927831798791885, 0.09771716594696045, 0.005457383114844561, 0.5045061111450195, -1.3549169301986694, 0.3274286985397339, -0.2618817389011383, 0.6757761240005493, -2.6267755031585693, 0.8997073173522949, -1.4892387390136719, -0.3290741443634033, 0.16219410300254822, -0.3183106482028961, -0.4214091897010803, -0.7407478094100952, 0.5280996561050415, -0.21518021821975708, 1.5319466590881348, -2.194950819015503, -0.10655223578214645, -1.446185827255249, -0.5226461887359619, -0.3249606490135193, 1.898162603378296, -0.3882076144218445, 0.79190593957901, -1.4497907161712646, -1.1731003522872925, 0.22683283686637878, -1.5082612037658691, -1.0986741781234741, -2.135762929916382, -1.7451900243759155, -0.012875430285930634, -0.821349561214447, 1.6775805950164795, 0.5513418316841125, 0.06586851179599762, -2.054985523223877, 0.10655619949102402, 0.15606586635112762, -0.6985884308815002, -0.39003893733024597, -1.1240581274032593, -0.3556614816188812, -0.5589010715484619, 0.6985534429550171, 2.1223554611206055, -0.6687219142913818, -0.05275646224617958, 0.6654011011123657, 0.8719230890274048, 1.9472163915634155, -0.8040953278541565, -1.2230660915374756, 0.5300139784812927, 0.015708358958363533, -1.2773305177688599, 0.17403344810009003, 0.33528655767440796, 0.44462403655052185, -1.2348345518112183, 1.619915246963501, 0.060232505202293396, -0.18816466629505157, 0.13282093405723572, -1.263784408569336, 2.9301629066467285, -0.16553305089473724, 1.1900049448013306, -0.21810919046401978, -0.5460704565048218, -0.29838740825653076, 0.7699726819992065, 0.4904254972934723, -0.6674764752388, 0.04952104017138481, -0.053390637040138245, 0.9813075065612793, -0.5326261520385742, 0.5410289764404297, -1.6626046895980835, -0.3747744858264923, 1.5958064794540405, -0.8170250654220581, -0.39909207820892334, -2.0387258529663086, -0.020504474639892578, 0.48221439123153687, 0.096771739423275, -1.5095558166503906, 0.09182199090719223, 0.2399439811706543, -1.3600777387619019, -1.5256569385528564, 1.0974334478378296, -0.09005942940711975, -1.583030343055725, 0.20727112889289856, -1.9409527778625488, -0.7422318458557129, -1.733941912651062, -1.1489168405532837, 1.920853614807129, 0.28228139877319336, -0.08132121711969376, 0.020979877561330795, 0.03755569085478783, 0.09360458701848984, 0.5390734672546387, 0.4910702109336853, -0.9492914080619812, -1.9105982780456543, 0.26699015498161316, 0.06082399934530258, 1.138668417930603, 0.6764362454414368, 1.024393916130066, -1.572090983390808, -0.6830836534500122, -0.46962401270866394, 0.5430772304534912, 0.5676925182342529, -1.3813210725784302, -0.8497657179832458], [-0.05868800729513168, 1.1091914176940918, -0.18489030003547668, 0.07922805845737457, 0.5863408446311951, -0.04712669923901558, 0.45088064670562744, -0.918777346611023, -0.7121881246566772, 0.6716106534004211, 0.2853718101978302, -1.9900108575820923, -0.29889899492263794, 0.4770342707633972, 1.3645455837249756, 0.3179137408733368, 0.21832506358623505, -1.711706519126892, -1.7723716497421265, -0.00901342649012804, 0.6174103617668152, -0.3918097913265228, 0.6474351286888123, 0.13786455988883972, 0.08038369566202164, 0.3930474519729614, 3.197639226913452, 1.954878568649292, -0.9523931741714478, 0.20379047095775604, -1.248695731163025, 0.5487001538276672, -0.7918563485145569, -0.27693212032318115, 0.8436934947967529, -0.10159535706043243, 0.1580624282360077, 1.1300201416015625, 1.1200743913650513, 0.44685500860214233, 0.6112799644470215, -0.7812577486038208, -1.2716578245162964, -1.2415518760681152, -0.7928018569946289, 0.08437050879001617, 0.47734928131103516, 1.2602835893630981, 0.9379948377609253, 0.4590969681739807, 0.5377900004386902, -0.3166789412498474, 0.9282041192054749, 0.3637009561061859, -0.40975889563560486, 0.08787326514720917, -0.03765091300010681, 0.3214057683944702, -0.13260988891124725, -1.441962718963623, -0.14693133533000946, 0.06714805215597153, 1.211294412612915, -1.6524945497512817, 1.96066415309906, -1.8287304639816284, -0.23299042880535126, -2.7112340927124023, 0.8942875266075134, 0.6700860857963562, 0.8679805397987366, 1.787911295890808, 0.054664645344018936, 0.5916575193405151, -0.7500579953193665, -0.3521776795387268, 0.23699431121349335, 0.8782930970191956, 1.1199795007705688, 0.6308325529098511, -0.14016997814178467, -1.2279404401779175, -0.871725857257843, 0.23924578726291656, 0.9626104235649109, 0.5565330386161804, -2.1599369049072266, -0.2642441391944885, -0.33876729011535645, 1.4381293058395386, 1.0215338468551636, 0.16418218612670898, -0.018866466358304024, -0.6785504221916199, 0.845726728439331, -1.234818935394287, -0.9953842759132385, -0.32422512769699097, 0.6328922510147095, 0.07698510587215424, 1.2664225101470947, -1.849002718925476, -0.9790756702423096, -1.5367748737335205, 1.4668476581573486, 1.1842284202575684, -0.15520349144935608, -0.41769737005233765, 1.1987009048461914, 0.044969066977500916, -0.0893770307302475, -1.1261446475982666, -1.4646601676940918, -0.5855401158332825, 0.7582114338874817, -0.7983170747756958, -0.028403211385011673, 0.9101261496543884, -0.4784810245037079, 0.3534706234931946, 0.07309792190790176, 1.9306204319000244, 1.5604090690612793, -1.3271195888519287, 0.5188176035881042, -0.7655264735221863, 0.012894729152321815, 0.660075306892395], [-1.73113214969635, 0.9182468056678772, -0.42763015627861023, -0.6173393130302429, -0.9205652475357056, -0.2135312706232071, -1.6030648946762085, -1.6993334293365479, -1.6666438579559326, 0.6502441763877869, -1.8252981901168823, 1.7013840675354004, 0.5191529989242554, -1.4165153503417969, 1.8593806028366089, 0.5750922560691833, 0.3376636207103729, -0.6063831448554993, 0.5051007866859436, 0.4164871275424957, -0.6540672779083252, -1.44746732711792, -0.20553486049175262, -0.07800283282995224, -0.401770681142807, -0.16402781009674072, 0.41848862171173096, 0.09484673291444778, 0.9992403984069824, -1.3405808210372925, 0.25955304503440857, 0.49977850914001465, 0.3276827335357666, 0.3002573251724243, 0.30047526955604553, -0.18430273234844208, -0.9391640424728394, -1.679551124572754, -0.4042257070541382, 1.6228039264678955, 2.4572227001190186, -0.8420253992080688, -1.801975965499878, -2.484201192855835, -0.017705164849758148, 0.5933727025985718, 0.8255995512008667, -0.057825036346912384, 1.5705279111862183, 1.3362282514572144, 1.2855477333068848, 0.7517538666725159, -1.0262762308120728, 0.7285793423652649, -0.2876608073711395, 0.2730899453163147, -1.5350360870361328, 0.22762486338615417, -1.7508783340454102, -0.16282369196414948, 0.7272085547447205, -0.36956632137298584, 0.5070778131484985, -0.11250805109739304, -0.42117998003959656, -0.3586413562297821, -0.6957423686981201, 1.5132321119308472, 0.025963008403778076, -0.6174997687339783, -0.3118928074836731, 0.11811242997646332, 0.29267555475234985, -0.5879919528961182, -1.2254149913787842, -0.19695721566677094, 0.11915203183889389, -0.4733143448829651, 0.48358845710754395, -0.5163387060165405, -1.7726625204086304, -1.2369955778121948, -0.44152429699897766, -0.8263751268386841, -1.5737732648849487, 1.2529869079589844, -1.8096545934677124, -1.5789698362350464, 1.016574501991272, -0.9318122863769531, 1.5877450704574585, -0.4444223642349243, -0.694972813129425, -0.5815635323524475, -0.48712873458862305, 0.03542783111333847, 1.161463737487793, -1.2172784805297852, -0.12329138070344925, -0.6226964592933655, -0.5031960606575012, 1.6146706342697144, 0.8262937068939209, 0.8398641347885132, 0.7056581377983093, 0.2537931501865387, -0.4511532485485077, -0.6328487396240234, 0.5647988319396973, -0.4298190176486969, 0.8099903464317322, -1.5246943235397339, 0.21832962334156036, 0.12393146753311157, -0.42097732424736023, 1.013255000114441, -0.10807555168867111, -0.6076095104217529, 0.2513545751571655, -0.07175663858652115, 0.49729543924331665, 0.5512686371803284, 0.2857457995414734, -0.5099329352378845, 0.8807488679885864, -0.6052001714706421, -0.3656797409057617, 0.1403171867132187]], [[0.7092687487602234, -0.20320168137550354, -0.13863083720207214, -0.37546804547309875, -1.249456524848938, -0.6568861603736877, 1.0954967737197876, -0.5365690588951111, 0.24087096750736237, -0.25321006774902344, 1.0411126613616943, 0.583898663520813, 0.7184091210365295, 0.4064091444015503, 0.32098352909088135, -1.7598485946655273, 0.5914009213447571, -0.2766363322734833, 0.6142681837081909, 1.00663423538208, 0.08557688444852829, 1.0208574533462524, 0.12040821462869644, 0.14717265963554382, -0.3936643898487091, 0.7573256492614746, 0.3983360826969147, 1.9875051975250244, -0.15326568484306335, -0.9855322241783142, -2.0332961082458496, 2.151533842086792, -1.106381893157959, -0.899321436882019, -0.04115454852581024, 0.7922659516334534, 0.18548734486103058, 1.0591145753860474, -0.3674286901950836, 1.7463054656982422, -0.6341281533241272, 0.5830602645874023, -0.20996329188346863, 0.4421499967575073, -0.7769969701766968, -0.826455295085907, 1.0072044134140015, 0.028777586296200752, -0.7365496754646301, -0.6861546635627747, -1.1443592309951782, 0.25367558002471924, -0.5054070353507996, -0.40341028571128845, -0.06150161102414131, -0.9223123788833618, 2.3483035564422607, -1.6042289733886719, -0.13639815151691437, 1.3023484945297241, 0.07024402916431427, 0.8715518712997437, -2.8833134174346924, 0.1200525090098381, 2.308652877807617, 0.7305607795715332, 1.4148743152618408, -0.2131415158510208, -0.5164387822151184, -0.3882403075695038, -0.8256806135177612, -0.6599619388580322, -0.5695491433143616, -0.8268934488296509, 2.459015130996704, -0.5348793864250183, -0.2463926076889038, -0.09994359314441681, -1.858258605003357, -0.14749523997306824, 0.023122450336813927, 0.3655782639980316, 0.8459494113922119, -2.1575610637664795, -0.8332693576812744, 0.06472863256931305, 2.0015671253204346, 0.8745917677879333, -0.2121133804321289, -0.2542386054992676, -0.3003354072570801, 0.7807336449623108, 2.8435122966766357, -0.1767987757921219, -0.2622232437133789, 1.661075472831726, 0.1484222710132599, -1.5775527954101562, -0.9100218415260315, 1.313305377960205, 0.7669659852981567, -0.0805722326040268, 1.9393846988677979, 0.9587033987045288, -0.3843742311000824, 1.2427419424057007, 1.5810067653656006, -0.5318490266799927, 1.3557686805725098, -0.03794525936245918, -1.221608281135559, 0.8773949146270752, 0.37095898389816284, 2.0778207778930664, 0.1705092191696167, 0.03216032311320305, 2.126124143600464, -1.2635846138000488, 0.6413781046867371, 0.7216143012046814, -0.14232677221298218, -1.6269237995147705, -0.43613147735595703, 0.301547110080719, -1.755039930343628, -1.096153974533081, 0.8912754058837891, 0.40076354146003723], [0.400269091129303, 0.26610684394836426, -1.1648823022842407, -2.348397970199585, 1.7886582612991333, 0.4784895181655884, -1.308014154434204, 1.109972357749939, 0.5090744495391846, 1.159896969795227, 1.766268014907837, 0.5009715557098389, -2.2467315196990967, -0.67254239320755, -1.3214499950408936, -1.0509264469146729, -0.6697022318840027, -0.09912795573472977, 1.0333831310272217, 1.4180799722671509, 0.6320785880088806, 0.19267535209655762, 0.8558760285377502, 0.1195460706949234, 1.0980212688446045, -0.5019252896308899, 0.05488855764269829, -0.3101898431777954, -0.10588382929563522, 0.9034163355827332, 2.834239959716797, -0.629165530204773, -0.9083101749420166, -0.21641720831394196, -1.158443570137024, 0.1405792236328125, -0.718981146812439, -0.6357595920562744, -0.27795785665512085, -0.9939441084861755, -0.12232191115617752, 0.32127732038497925, -0.24037636816501617, -0.6110310554504395, -0.11092615127563477, 0.5969594120979309, 0.591119110584259, -0.7346266508102417, -0.6995800733566284, -0.5479962825775146, -0.22476403415203094, -0.08080621063709259, -0.11418921500444412, -0.3500964343547821, 0.4716379940509796, 0.1864464282989502, 0.4351588785648346, 0.46544933319091797, 0.9956181049346924, 0.08393270522356033, -1.3845049142837524, -1.104371190071106, -1.2717221975326538, -0.16020843386650085, 0.843513011932373, 0.8392074704170227, -1.7605501413345337, -0.2641998529434204, 0.8878825902938843, -1.0401862859725952, -0.40634557604789734, 1.566533088684082, -0.4609215259552002, -0.6716131567955017, -1.55593740940094, -2.2213947772979736, -1.3831977844238281, -0.7481151223182678, -1.4531047344207764, -1.5395718812942505, 0.23886363208293915, 0.7916027307510376, 1.3894636631011963, -0.8463796973228455, 0.05532638728618622, -1.015317440032959, 0.23495739698410034, 0.9587277770042419, 1.132332444190979, -1.906647801399231, -0.08552248030900955, -1.4656720161437988, -0.3575047552585602, 0.7656615376472473, -0.6089736819267273, 1.2185055017471313, -0.11090577393770218, -0.2981577217578888, 1.0017253160476685, 1.4690732955932617, -0.6537963151931763, -1.5031554698944092, -0.8573671579360962, 0.3333317041397095, -1.4121991395950317, -0.19817940890789032, -0.09791368991136551, -0.5106222033500671, -1.8337395191192627, -0.06381623446941376, -0.21107204258441925, 0.31669673323631287, -0.20443519949913025, 1.4293590784072876, -1.2973588705062866, -0.4766887128353119, -0.9099008440971375, 1.0954904556274414, -1.3791996240615845, -0.2891206741333008, 0.8936399817466736, 0.47740665078163147, 1.4088809490203857, 0.08734960108995438, 0.3269284665584564, -0.25886431336402893, 0.05775168165564537, 0.3192734122276306], [-0.03075260855257511, -0.42660441994667053, -1.5313642024993896, -0.43351882696151733, 1.3349361419677734, 1.8064254522323608, -0.7864043116569519, 0.3760395348072052, 0.9474992156028748, 2.1414270401000977, 0.3374793529510498, 3.001293897628784, 0.06972507387399673, -0.358016699552536, -0.9813661575317383, 0.382296621799469, 1.2562837600708008, 0.202593594789505, 0.5228489637374878, -2.539794921875, -0.8345608711242676, 0.4588116705417633, -0.35683131217956543, 0.08301027864217758, -0.14668558537960052, 0.0525248683989048, 0.7597121000289917, -1.2109582424163818, -0.7776340842247009, -0.290404736995697, 1.7846553325653076, -1.6162723302841187, -0.5106021761894226, 0.9680299758911133, 0.6212196946144104, 0.7463685274124146, 0.42683473229408264, 0.18067730963230133, -0.09401005506515503, 0.6303634643554688, -0.26279473304748535, 0.7867183089256287, -1.2819702625274658, 2.0853097438812256, 1.4893519878387451, 0.5429922938346863, 0.36617377400398254, 0.012631328776478767, 0.4731198847293854, 1.396546483039856, -0.6754907965660095, 1.1165047883987427, 0.4840319752693176, 0.7560634613037109, 1.0880428552627563, -1.1110526323318481, 1.5659948587417603, 0.07575690001249313, -0.9521670341491699, 0.5964251756668091, 0.10509762167930603, -1.9197123050689697, 0.2340039759874344, 1.6042168140411377, 0.06078249216079712, 0.09824126958847046, 1.291195034980774, 1.1901146173477173, 0.30190950632095337, -0.6137492656707764, -0.010142873041331768, 0.5239965915679932, 0.018358150497078896, -0.6955366134643555, 0.6749096512794495, 0.9630789756774902, 1.11224365234375, -1.330153465270996, 0.013473420403897762, 0.1817253828048706, -0.7544295191764832, 1.3363087177276611, 0.08779285848140717, 0.9525443315505981, 0.6533162593841553, -0.2606690526008606, -1.5144973993301392, -0.48713207244873047, 1.1075152158737183, -1.6562206745147705, -1.0106685161590576, -0.40076810121536255, 0.7896002531051636, -1.6429136991500854, -0.3504306674003601, 0.0396369993686676, 1.1736470460891724, 0.021533675491809845, 0.8272342681884766, -0.08490483462810516, 0.6838119626045227, 0.6513735055923462, 1.0174793004989624, 0.2050531804561615, 0.13132093846797943, -1.0922361612319946, 2.707157611846924, -2.598557710647583, -0.22901524603366852, 0.9571366310119629, -0.5466598272323608, 1.0493285655975342, -0.7754207849502563, 2.2310683727264404, -0.0317552424967289, 0.7937690019607544, 0.9440647959709167, -0.9709541201591492, 1.9538761377334595, 0.20119015872478485, 0.12062931060791016, 0.8080053329467773, 1.2307389974594116, 1.1127746105194092, -0.003142997156828642, -1.1604524850845337, -0.28426113724708557, 0.08360755443572998], [1.9923717975616455, 0.7983180284500122, 0.5925917029380798, 0.6653943657875061, 0.2743847668170929, 0.5733282566070557, -1.2200273275375366, 0.11895608901977539, 0.21299999952316284, -0.7409311532974243, 0.5789064168930054, 0.2927126884460449, 0.6206923723220825, 1.3821194171905518, -0.9716208577156067, -0.05422770977020264, -0.4475657045841217, 0.22486789524555206, -0.08205126971006393, 0.18673300743103027, 1.0624854564666748, 0.23705297708511353, 1.5052417516708374, -0.13933324813842773, -0.20901598036289215, -0.7197533249855042, 1.0468239784240723, 1.0138137340545654, -0.05345411226153374, 0.7927621006965637, 0.15461871027946472, -0.4387294054031372, 1.4790806770324707, -0.11084813624620438, -0.578427255153656, 0.8587244153022766, 1.7287145853042603, -1.1012414693832397, 0.7822652459144592, 0.20326267182826996, 3.4748146533966064, -0.022024381905794144, -0.15434904396533966, 0.8774604797363281, -1.8655989170074463, 0.2728821337223053, 1.2904311418533325, -1.5655338764190674, -0.3661181628704071, 0.041544996201992035, -0.42720475792884827, 1.3755035400390625, 1.1955970525741577, 0.6390146017074585, -0.8184658288955688, 1.7927478551864624, -1.8814870119094849, -0.29634079337120056, -0.8462123870849609, -0.0676708072423935, -1.0723482370376587, 0.40866291522979736, -1.2097395658493042, 0.4266221523284912, -0.3355658948421478, 0.11175768822431564, 0.32929620146751404, 1.3938241004943848, -1.8521534204483032, -1.3984434604644775, -0.04713795706629753, 0.35686516761779785, 0.45746544003486633, 1.6806477308273315, 0.1345389485359192, 1.0693936347961426, 0.8354633450508118, 1.2422128915786743, -0.11632636189460754, -0.5163020491600037, 0.5529488325119019, 0.981134295463562, 0.30810871720314026, -0.1780373752117157, 0.47010570764541626, 0.867129921913147, -0.4292084276676178, 1.3400534391403198, 2.053988456726074, 0.2974611818790436, -0.99294114112854, -0.046147529035806656, 0.05433515086770058, 1.544084906578064, -0.5397338271141052, 1.031071424484253, 0.10766074061393738, -0.5201870203018188, 0.8314236402511597, 1.5171711444854736, 1.7832767963409424, 0.5826530456542969, 0.7687479257583618, -0.43186265230178833, 1.250603437423706, 2.261997938156128, -1.9206944704055786, -0.04098997265100479, -1.230965256690979, 1.6962549686431885, -2.311513900756836, -0.3189079463481903, 1.2417829036712646, -0.6582978963851929, 1.7919716835021973, 1.0631954669952393, 0.5159764289855957, 0.5727049708366394, -0.3425382375717163, 0.23475171625614166, 0.2547052502632141, -0.01394769735634327, 0.8729797005653381, -1.424687385559082, 1.3766827583312988, -0.8679646849632263, -2.7549867630004883, 0.4183976352214813], [0.33528363704681396, 0.047793351113796234, 0.5136117935180664, 0.6353681683540344, 0.34559857845306396, -0.8504624962806702, 0.0830734595656395, 1.2580513954162598, 0.8527936339378357, 3.0422279834747314, -1.6874442100524902, 0.17994697391986847, -0.4295074939727783, 1.471693754196167, 1.539783000946045, 0.9046091437339783, 0.353059321641922, -0.47473451495170593, -1.2584929466247559, -0.4661886990070343, 1.2343276739120483, -0.5839711427688599, 0.6592716574668884, -1.099979043006897, -1.3494341373443604, -1.5763561725616455, 0.4159323573112488, -2.3725967407226562, -0.3832431435585022, -0.762843132019043, 0.8043279647827148, -1.555602788925171, -2.0964138507843018, 1.5037305355072021, 0.7000661492347717, 1.4404451847076416, -0.5045846104621887, -0.35357287526130676, 0.20900669693946838, -0.4769305884838104, 0.8355482816696167, -0.5385423302650452, 1.6243267059326172, -0.031613271683454514, -0.5377919673919678, -0.13659648597240448, -0.1707560420036316, -1.4420243501663208, 0.21790294349193573, 0.6686450839042664, 0.04008670523762703, -1.1724389791488647, -1.6198275089263916, -0.30293720960617065, -0.9678935408592224, -0.8439143896102905, -2.197222948074341, -0.13172070682048798, -1.057737946510315, -0.7226272225379944, 0.7893511056900024, -0.411845326423645, -0.2956972122192383, 0.313845157623291, -2.8456664085388184, 1.1974835395812988, -0.34913522005081177, -0.21449467539787292, 1.3213680982589722, -1.3267415761947632, 1.305187463760376, -0.2735164165496826, 1.23294198513031, -0.3095664381980896, -0.4156959652900696, 0.339409202337265, -1.0522536039352417, -0.7258356213569641, 1.1433011293411255, 0.43972957134246826, 1.5505485534667969, -0.5169699788093567, 1.471024513244629, -1.412643313407898, -1.4770252704620361, 0.6595693826675415, -0.7749794125556946, 0.6785833239555359, 1.1385465860366821, 1.4135833978652954, 0.48778852820396423, -0.15984757244586945, 1.1961891651153564, -0.20351780951023102, 0.5475124716758728, -0.7009292244911194, -1.1351661682128906, -0.17250923812389374, -0.006600753869861364, -1.6952470541000366, 0.5012947916984558, 0.7269221544265747, -0.8671849966049194, -0.2231549322605133, 0.13462069630622864, -0.9543787837028503, 0.32038626074790955, 0.07882005721330643, 1.1091145277023315, -0.3286665081977844, 0.4821489453315735, -0.38551566004753113, 0.35328254103660583, 0.4560815095901489, 1.417763113975525, -0.27125072479248047, -0.2994183301925659, 2.5846176147460938, 0.3108341097831726, 1.786159873008728, -0.009537240490317345, -0.3916361629962921, 0.4712553024291992, 1.2440578937530518, -0.4646663963794708, -1.315738320350647, 2.226840019226074, 0.282587468624115], [-0.880725085735321, 0.34459638595581055, 0.9919037818908691, 0.0798708125948906, -1.435280680656433, 0.19128084182739258, 0.262112021446228, -0.8475600481033325, 1.6327807903289795, -1.5776993036270142, 1.4583508968353271, 1.2160255908966064, 1.0182323455810547, 1.7072316408157349, -1.4611949920654297, -0.8312875628471375, 1.344992756843567, -0.4623669683933258, -1.652920126914978, 0.46039828658103943, -0.49249380826950073, -0.7727670669555664, 0.5636411309242249, -1.0980561971664429, 0.12662723660469055, -0.4309983253479004, -0.0733838826417923, -0.2852036654949188, 0.02055881731212139, -0.8984483480453491, 0.2906230688095093, 0.6092696189880371, 0.463331013917923, -0.26131975650787354, -0.1889367550611496, -3.296269416809082, 0.3200463056564331, -1.5550402402877808, 0.27143406867980957, -0.4426671266555786, -0.2471175491809845, 1.4525034427642822, -0.33267736434936523, 0.7299202084541321, -0.21056875586509705, -0.0021759173832833767, -0.29399290680885315, 0.0071929702535271645, 0.8009925484657288, -0.5348021984100342, 0.4376990795135498, 0.5527145862579346, -1.2207152843475342, 0.4461286664009094, -0.7682747840881348, 0.13197383284568787, 0.6816995739936829, -1.8715832233428955, -1.6308971643447876, 1.4155921936035156, 1.4039461612701416, 1.3717013597488403, 0.5837738513946533, -1.4950120449066162, -1.9050837755203247, -1.661252737045288, 0.8930780291557312, -0.2559491693973541, -0.42065364122390747, 0.8269228339195251, -0.47699394822120667, 0.051318854093551636, 0.27660489082336426, 0.1652233749628067, -0.7747721672058105, 0.6435398459434509, -0.014987933449447155, 1.309999704360962, -1.4325149059295654, -0.35889512300491333, -0.16745366156101227, -0.6089552640914917, 0.6632524132728577, -1.0306239128112793, -1.6702498197555542, -0.5365027785301208, 0.6556593179702759, -1.001185655593872, -0.021506115794181824, 0.7531833052635193, -2.7631328105926514, 0.0711163729429245, 0.5860000252723694, -0.8575341105461121, 0.18749703466892242, -1.1681491136550903, 0.32701990008354187, -0.6401777267456055, 0.8989960551261902, -1.4772415161132812, -2.222511053085327, 0.18576034903526306, -0.5477967858314514, -0.1580718606710434, 0.9491028785705566, 0.14375144243240356, -1.2635120153427124, -0.023001575842499733, -1.073399305343628, -0.3580402731895447, -1.5129845142364502, 0.5308614373207092, 0.3248094320297241, 0.2264041155576706, 0.4677886962890625, 0.7231842279434204, 0.12660394608974457, 1.7963590621948242, 1.9443466663360596, 0.885473370552063, 0.9455410838127136, -1.4032819271087646, 1.0132029056549072, -1.2785382270812988, 1.2870244979858398, -0.16818922758102417, 0.2744351625442505, -0.3173532783985138], [-0.7192097306251526, 0.06485041230916977, -0.948416531085968, 0.14880506694316864, 2.3506367206573486, -0.6625604033470154, -0.35424843430519104, 0.09027887135744095, 0.2516613304615021, -0.9209667444229126, 1.3441728353500366, -0.10774378478527069, 1.0739787817001343, -0.08645326644182205, 0.6840302348136902, -0.7563583850860596, 0.6987307071685791, -0.4401525855064392, -0.44016391038894653, -1.1690884828567505, -1.8094477653503418, 2.225067615509033, 0.28656259179115295, 0.3750441372394562, 0.6322923898696899, -1.7460485696792603, -0.4462166130542755, -0.9854358434677124, 0.18348418176174164, -0.15811720490455627, -1.005250096321106, 0.945395827293396, -0.09356493502855301, 0.47707244753837585, 0.2276225984096527, -0.19578424096107483, -0.7276831269264221, -0.25283288955688477, -0.004343811422586441, -0.6074845194816589, -0.10086548328399658, 0.4537847340106964, -0.8660979270935059, -0.8022370338439941, -0.7643497586250305, 1.3569982051849365, -0.5405898094177246, -1.1839263439178467, 0.7262800931930542, -2.014338970184326, 0.11351785808801651, -1.920279860496521, 0.39430883526802063, 0.6098207831382751, -1.4631168842315674, 0.2094542384147644, 0.0964709222316742, 1.6636682748794556, -1.2955491542816162, 1.4059070348739624, -0.2476608157157898, -0.45872169733047485, -0.7514818906784058, -1.7165645360946655, 2.737445116043091, 0.6175253987312317, 1.4766428470611572, -0.9460335373878479, -0.1251029670238495, -1.7387546300888062, 0.06854679435491562, 1.1518433094024658, 0.02943960763514042, 1.4872758388519287, -0.6164153814315796, -0.548712432384491, 0.21091009676456451, 0.4465300142765045, -0.7978692054748535, -0.21875537931919098, 1.2931872606277466, 0.6090320944786072, -0.31008660793304443, 1.2693381309509277, 0.6849637031555176, -0.11772213131189346, -0.037101782858371735, -0.3779795169830322, -0.4421440064907074, 0.687267541885376, -1.6790494918823242, 0.5950948596000671, -0.3208921551704407, -0.5370891094207764, -0.5903297066688538, -0.22617116570472717, 1.076697826385498, 0.2778981626033783, -0.5476650595664978, 0.11938177049160004, 0.835579514503479, 0.5460296869277954, -1.8525127172470093, -0.49513140320777893, -1.4032375812530518, 0.36054733395576477, 1.4782230854034424, 1.3861441612243652, 0.4378153383731842, -0.4771062731742859, 2.337143898010254, -0.4695417582988739, -0.812250554561615, -2.0102739334106445, -0.9899458885192871, -2.4080657958984375, 0.3112933039665222, -1.5683135986328125, 0.0928293988108635, 0.6555100083351135, -0.7589622735977173, -0.35117146372795105, 1.347668170928955, -0.31909582018852234, -0.6307259798049927, -1.0751338005065918, 0.2500764727592468, 0.6716048717498779], [-1.3114763498306274, -0.9545655250549316, -0.962153971195221, 0.9868611097335815, 2.0527403354644775, -0.7147455811500549, 0.9471287131309509, 1.1730138063430786, -1.0999207496643066, 1.5268750190734863, -1.2619373798370361, -1.1691455841064453, -0.535643458366394, 1.4136865139007568, 1.3734627962112427, 0.783497154712677, 1.408657193183899, 0.2853085398674011, -1.2079294919967651, -0.4913545846939087, 0.3572171628475189, 0.23780004680156708, -0.8709983229637146, -0.9822015762329102, -1.2392090559005737, -0.6455107927322388, 0.17953623831272125, -1.9151731729507446, -0.7725074887275696, -0.39041826128959656, -0.6481088995933533, 0.15682554244995117, -0.15366418659687042, -1.783686876296997, 1.4437464475631714, -1.2781496047973633, -0.6757705211639404, -1.1552150249481201, -0.2823825478553772, 0.5631542205810547, -1.4248305559158325, -0.6727563142776489, -1.115440845489502, -1.5503581762313843, -0.43201780319213867, -2.1428353786468506, 1.1054373979568481, 0.14072264730930328, 0.4662769138813019, -0.07662920653820038, 0.025871187448501587, -0.7764281630516052, -1.337964415550232, -1.9752694368362427, -0.7247486114501953, -0.6954493522644043, -0.3440472483634949, 0.42873236536979675, 1.298520565032959, 1.780366063117981, -0.1732066571712494, -0.06129538640379906, 0.2786940038204193, 0.6892370581626892, -1.7463035583496094, -0.08476225286722183, -1.2199450731277466, 0.5986002683639526, 0.12791596353054047, 1.40630042552948, 1.3987154960632324, 1.5094528198242188, -0.4896574020385742, -0.8984569311141968, 1.7970695495605469, -1.7723942995071411, -1.1889787912368774, -0.8819541931152344, 0.4315066635608673, 1.0036112070083618, 1.6185827255249023, -0.6299558281898499, -0.7694566249847412, 0.9896828532218933, -0.4447917342185974, 0.4595760405063629, -0.22570642828941345, -0.849991500377655, -0.5953689217567444, -0.8251193761825562, 0.30984506011009216, 0.1374569833278656, -0.1939551830291748, 0.40308839082717896, 0.7350822687149048, 0.33390966057777405, 0.5476318001747131, 0.6320378184318542, -0.826095700263977, 0.08147622644901276, -1.4708315134048462, -1.0108989477157593, -1.3359031677246094, -0.7151052355766296, -0.4238924980163574, -0.5831649303436279, -0.3896857798099518, -0.2569648325443268, 0.22922749817371368, -0.1311822384595871, 2.4144399166107178, 0.08795411139726639, 0.8136783242225647, 0.021664926782250404, 0.2463984489440918, -0.0072837891057133675, 1.29989755153656, -2.6729986667633057, 1.1500442028045654, 0.2958393692970276, -0.44115495681762695, 1.9204986095428467, -2.494903087615967, 0.9936369061470032, -0.4084477722644806, 0.08203588426113129, 1.0288984775543213, -0.15677061676979065], [1.2542657852172852, -1.1314655542373657, 1.4904589653015137, 0.9183121919631958, -1.774707317352295, 1.7859748601913452, -0.3905893862247467, 0.17084889113903046, 0.6201527714729309, 0.04110288247466087, 0.42763397097587585, 1.6019501686096191, -0.3970277011394501, -1.2884513139724731, -0.4103486239910126, -1.3541442155838013, 0.20465701818466187, -1.121226191520691, 0.7013086080551147, 0.5079914927482605, 1.0504056215286255, 0.41274866461753845, 0.11189696192741394, -1.7117441892623901, 1.6244744062423706, -1.7228035926818848, 0.1486036479473114, 0.3020440638065338, -0.7035570740699768, -0.3226918578147888, -0.33732983469963074, 0.25865864753723145, -1.5051895380020142, -0.41120901703834534, -1.125134825706482, 0.736214280128479, 1.1302950382232666, -1.0765012502670288, -1.8540862798690796, 0.3980903625488281, 0.48694008588790894, 1.2704812288284302, 1.798236608505249, 0.1259712427854538, -0.646588146686554, 0.09050221741199493, -0.322824627161026, -0.7179615497589111, -0.78254234790802, 0.17343968152999878, -0.8128791451454163, 0.7248539328575134, 0.33263877034187317, 0.09078828245401382, 1.0402462482452393, -1.4301851987838745, -1.334895133972168, 0.8227076530456543, 1.9791452884674072, -0.27147915959358215, -0.8909919857978821, 1.8775790929794312, 0.3430365025997162, -0.9132824540138245, -1.1292002201080322, 0.26046180725097656, 0.09857209026813507, -0.5858811140060425, -0.8737637400627136, -1.7182587385177612, 1.068919062614441, 1.0584304332733154, 0.9058506488800049, -0.46065691113471985, 1.1726493835449219, -0.6626402139663696, -0.6059598922729492, 1.3021132946014404, -0.3367089033126831, -1.5757697820663452, 0.3026125431060791, 0.9899929165840149, -0.08750467747449875, -2.333796262741089, 1.2497928142547607, 0.08703631162643433, 0.7698201537132263, -1.7193604707717896, -0.7059505581855774, 0.5547540783882141, -0.4895179569721222, -0.16565145552158356, -0.03676048666238785, 0.5992739200592041, -1.1725354194641113, 0.1958693116903305, -0.7409565448760986, -1.0412877798080444, -0.3735387921333313, 0.16767384111881256, 0.7302685976028442, 1.1264950037002563, 0.3259781301021576, 1.2661736011505127, 2.493008613586426, -0.5642350912094116, 0.3812662959098816, -0.3854677677154541, 1.6021462678909302, -0.8556393980979919, 0.8738681077957153, 0.37690678238868713, -1.0525388717651367, -0.5496441125869751, -0.7824406027793884, -0.6439560055732727, -0.8684948086738586, -0.5364643335342407, 2.3273656368255615, 0.606595516204834, -0.3977199196815491, -0.56801837682724, -1.2331253290176392, 0.6501784324645996, 0.7660275101661682, 0.3531704246997833, 1.148064374923706, 1.30326247215271], [1.7556712627410889, 0.27754709124565125, -0.564618706703186, 0.37296146154403687, 1.0629031658172607, -0.4765092432498932, 1.338579773902893, 2.1297144889831543, -1.8729861974716187, 2.615128517150879, -0.5214205384254456, 0.8277781009674072, -0.48812490701675415, 0.9409146308898926, 0.44103875756263733, -2.0145111083984375, -0.03730561211705208, 1.0395992994308472, 0.7822389602661133, -1.274876356124878, -1.3218605518341064, -1.149839162826538, -0.5918534398078918, 0.12347373366355896, -0.23883481323719025, -0.4799124002456665, -0.5177185535430908, -1.4250781536102295, 0.943020224571228, -0.5224285125732422, -0.5412615537643433, 0.68552565574646, -1.7276661396026611, -0.351447731256485, 0.5136277079582214, -0.04397653043270111, -1.8424019813537598, 1.8161673545837402, -1.3517370223999023, -1.3205394744873047, -0.3340688645839691, 0.01835050992667675, -0.055188123136758804, 0.9402804374694824, -0.07982238382101059, -0.6225053071975708, 0.6791321039199829, 2.0327677726745605, -1.2892236709594727, -1.3067522048950195, 0.9451497197151184, -0.8937757015228271, -0.9747244119644165, 0.4583772122859955, -1.4282232522964478, 0.9198121428489685, -1.3754112720489502, 0.8576692342758179, 1.0229154825210571, 0.6039249897003174, -0.668990969657898, 0.8691812753677368, -0.9728406667709351, -0.17120055854320526, -1.2790050506591797, 0.15332771837711334, 0.9320427775382996, 0.3325519561767578, 0.09431304782629013, -1.3251715898513794, -1.8749717473983765, 1.3616375923156738, 1.2470427751541138, -0.3570865988731384, 0.8416494131088257, -0.550366997718811, 0.34177252650260925, 0.3017260730266571, -0.5436804890632629, 0.5268807411193848, -2.0540387630462646, -0.640071451663971, -0.29859739542007446, 0.2475830465555191, -0.8509341478347778, 0.22222712635993958, -1.6135562658309937, -0.40189826488494873, -0.5076345801353455, 0.12513665854930878, -0.07409712672233582, 0.25710272789001465, -1.1773101091384888, 0.7875933051109314, 0.436920166015625, -2.7946462631225586, 0.9820879697799683, -0.2507959008216858, -0.4788317382335663, -1.44699227809906, 1.4849188327789307, -0.873692512512207, -0.7241269946098328, 1.0247787237167358, 3.1798276901245117, -1.0061415433883667, 0.032393716275691986, -1.3295488357543945, 0.5916672945022583, 0.3653275668621063, -1.2032697200775146, 0.35824504494667053, -0.8503055572509766, 0.9632182121276855, 0.10139956325292587, -0.5247623920440674, -1.43878173828125, -0.5383896231651306, -0.9594379663467407, 0.12438174337148666, 1.124313473701477, -0.19461020827293396, 0.42351287603378296, 1.3025258779525757, 0.9812647700309753, -0.18450263142585754, -0.5226386785507202, 0.06921349465847015], [-0.972160279750824, -0.3500429689884186, 1.1678305864334106, 1.5255000591278076, 1.021814227104187, 1.7842440605163574, 2.214503526687622, -0.11127366870641708, -0.6099086999893188, -1.347455382347107, -1.1611846685409546, 0.5052098631858826, -0.6897922158241272, 1.398290753364563, -0.47502225637435913, 1.8476569652557373, -1.2453923225402832, 0.5295153856277466, 0.041794177144765854, -1.4289723634719849, -1.0689706802368164, -1.1926466226577759, -0.031974878162145615, 0.6320939064025879, -1.0220764875411987, -1.0553805828094482, 0.31448662281036377, 0.7100869417190552, -1.0228767395019531, -0.8712555170059204, -0.12113222479820251, -1.2288758754730225, 1.754612922668457, 0.6528047919273376, -0.014494849368929863, -0.7753233313560486, 1.9297528266906738, 1.023201823234558, 1.0673155784606934, 0.8268429040908813, -1.9419548511505127, -0.297039657831192, -0.07204651832580566, -1.8376260995864868, 0.6955649852752686, 1.3366811275482178, 0.7053126096725464, -1.7145200967788696, -0.6096802353858948, 1.081678032875061, -1.844522476196289, -0.375810831785202, 0.9349539279937744, -0.7757200002670288, -0.5147177577018738, 1.531552791595459, 0.25156089663505554, 0.5176905989646912, 0.10323791205883026, 0.5586642026901245, -0.46934545040130615, 0.22428056597709656, -0.8346437811851501, 1.0047568082809448, 0.033746782690286636, -0.4497108459472656, -0.32016271352767944, 1.1483362913131714, -0.28025639057159424, -0.24758104979991913, -0.3071444034576416, -1.5154739618301392, 1.0897929668426514, -1.1276758909225464, 0.8038402795791626, -0.29263755679130554, 0.0411265604197979, 0.33958953619003296, 1.4407851696014404, -1.4380536079406738, 0.19638463854789734, 0.5613980889320374, 0.5088040828704834, -1.6684623956680298, -1.1062932014465332, 0.25165680050849915, -1.4818934202194214, -1.062313199043274, 1.6158424615859985, -1.6050355434417725, -0.7127230763435364, -2.2025842666625977, 1.5704524517059326, 1.8994609117507935, -0.5501918792724609, -1.578287124633789, 1.0398825407028198, 0.2777967154979706, -1.0408717393875122, 0.9517881870269775, 0.3036648631095886, 0.8248509168624878, 1.30094575881958, 0.19209028780460358, -0.7359718084335327, 1.9728868007659912, -1.2789993286132812, 1.0248576402664185, 1.6040774583816528, 0.6257633566856384, -0.7534840106964111, -0.4581840932369232, 1.7490440607070923, -2.1933155059814453, -0.10519971698522568, 1.4375088214874268, 0.19653227925300598, 2.3463218212127686, -2.1328744888305664, -0.11968038231134415, -1.058786153793335, 0.5861961245536804, -0.10388068854808807, 0.9434962868690491, -0.0524403378367424, 1.9519171714782715, -0.3633963465690613, -0.9916825294494629], [0.7391082644462585, 1.1126246452331543, 0.04059028625488281, -0.038232650607824326, 2.4023196697235107, 0.229835644364357, 1.2331572771072388, -1.6723904609680176, 1.7363941669464111, -0.7307560443878174, 1.5535749197006226, 1.0712697505950928, -0.8914204239845276, 0.023903673514723778, -0.45935848355293274, -0.3604285418987274, -0.0969395861029625, 1.462365984916687, -0.12854142487049103, -1.6157158613204956, 1.0654455423355103, -1.1640949249267578, -0.8567683696746826, 0.829622745513916, 0.6330804228782654, -0.07732705026865005, 0.7750499248504639, -0.6052854657173157, 0.959324300289154, 1.5727413892745972, -0.2594141364097595, -1.3042035102844238, 1.5083394050598145, -1.4878618717193604, 0.699971616268158, -1.1217679977416992, 1.1379350423812866, -0.4561820924282074, -1.5473411083221436, 0.8774161338806152, -0.9152786135673523, -0.1546088606119156, -0.5734562277793884, 0.4330156445503235, 0.2123880237340927, 2.6001946926116943, 0.980176568031311, -1.1248128414154053, -0.5294883251190186, -0.45901259779930115, 2.3247127532958984, -0.6631133556365967, -0.09834101051092148, -0.2536991238594055, -1.234980583190918, 0.8626635670661926, 1.897612452507019, 1.441976547241211, 1.1896240711212158, 0.7660006880760193, 1.8732938766479492, -0.07901076227426529, -0.9888986349105835, -0.990675151348114, -0.5304122567176819, 0.40051132440567017, 0.4515710771083832, 0.21452850103378296, -0.30745914578437805, -1.4308451414108276, -0.2633509039878845, -0.48529958724975586, -0.031694818288087845, -1.7530276775360107, 0.21181100606918335, 0.1863095611333847, -0.8973348140716553, 0.5362256169319153, 2.7685742378234863, 1.764744758605957, -0.6272173523902893, 0.4788566529750824, 0.6915478706359863, -1.0107691287994385, -0.7563648819923401, 1.6809512376785278, 1.0547494888305664, 0.12623564898967743, 0.13847440481185913, -0.481376975774765, 1.1369578838348389, 0.8602330088615417, -0.44062355160713196, 0.27811312675476074, 0.04435677453875542, -0.6695832014083862, 0.2787308692932129, -1.0268667936325073, -0.1870855689048767, 0.5036542415618896, 1.4589147567749023, -0.1857605278491974, 1.3492865562438965, -0.21863295137882233, 1.0543856620788574, 1.4813194274902344, 0.47946053743362427, -0.15864482522010803, -2.771566390991211, -1.717463731765747, 0.2007008194923401, -0.19369076192378998, -0.033072102814912796, 0.9733387231826782, -0.3469785749912262, 0.7027660012245178, 0.3216571807861328, -1.2951806783676147, -0.26051297783851624, 0.18741485476493835, 0.14522407948970795, 0.7733677625656128, 1.4385430812835693, 0.40826690196990967, -2.6148111820220947, -1.4702372550964355, 0.3964575231075287, 0.189729243516922], [0.5647487640380859, -0.4691148102283478, -0.8390646576881409, -2.119741439819336, 1.2697396278381348, -0.17913271486759186, -0.012074953876435757, 0.8081372976303101, 0.424050509929657, -1.169085144996643, -0.04846527427434921, -1.700931429862976, 0.5132476687431335, 0.3563413619995117, 0.3405965566635132, -0.11801651865243912, -0.6041409373283386, -1.4528546333312988, 1.2263071537017822, 1.3986080884933472, -1.8026334047317505, 1.262807846069336, -0.39605799317359924, 1.1857118606567383, 0.37867745757102966, 0.07788706570863724, -0.8049224615097046, 0.9496068358421326, -0.07573398947715759, -0.26977959275245667, -2.0265212059020996, 1.411461353302002, -0.5114925503730774, 1.3836973905563354, 1.2021013498306274, 0.749608039855957, 1.1505497694015503, -0.008827163837850094, 1.957452416419983, -0.927223265171051, -0.049312446266412735, -0.9200973510742188, 1.2908564805984497, 0.5107924342155457, -1.6432687044143677, 0.46641016006469727, -0.8996498584747314, 0.36936789751052856, -0.5529613494873047, -0.4179026782512665, -0.0038169475737959146, -1.763211727142334, -1.4787425994873047, 0.8576810956001282, -0.7199608087539673, -1.6171027421951294, -0.33151355385780334, -0.5633963942527771, 1.2076127529144287, -1.2096877098083496, 0.4218611717224121, 1.9306038618087769, -0.07640667259693146, -0.3393857479095459, 1.137536644935608, 0.7252888083457947, -0.9137760996818542, -0.0448981374502182, 0.12066970020532608, -0.7308655381202698, -1.2987067699432373, -1.0659195184707642, 1.9390053749084473, -0.31480517983436584, 0.2966500520706177, 1.7534304857254028, 0.4642014503479004, -1.4546979665756226, -1.1487960815429688, 1.3570466041564941, 0.7853780388832092, 1.0358108282089233, -1.2498401403427124, 0.3628290593624115, -0.4411779046058655, -0.08141235262155533, -0.5220603346824646, -0.2501715421676636, -0.9225900173187256, -0.16762197017669678, -0.02523145079612732, 0.33306244015693665, -0.6311191320419312, 0.6475819945335388, -0.4332675635814667, -0.61371910572052, 1.0635082721710205, 1.9768882989883423, -1.8447105884552002, 1.7256076335906982, 0.6751457452774048, -0.5249210000038147, -0.273090124130249, -0.23542393743991852, -0.3664928078651428, 0.5254719853401184, 0.2535121738910675, -1.0169936418533325, -1.7334189414978027, 0.7157883048057556, 1.1003113985061646, -0.7587859034538269, 0.4361893832683563, 0.8756173849105835, -1.7375901937484741, -0.06461154669523239, -1.296583890914917, 0.836936891078949, -1.3069474697113037, -0.9420373439788818, -0.23078875243663788, 0.2677282989025116, -1.2187206745147705, -0.7127789855003357, -0.18275976181030273, 1.6987073421478271, -0.5175496339797974, -0.15004238486289978], [-2.5340638160705566, -0.8149845600128174, -0.5489063858985901, -0.6109864711761475, 0.44678589701652527, -1.2176218032836914, 0.17137308418750763, 0.7398212552070618, -0.775063693523407, -0.07894667237997055, -0.2536522448062897, -0.10857672989368439, 1.1580547094345093, 0.1339738816022873, -1.72248375415802, -1.1829816102981567, -0.20083017647266388, 0.8591976761817932, 1.0517629384994507, -0.144046351313591, -1.2951463460922241, -0.026379121467471123, -0.016478614881634712, -1.2246863842010498, -0.885138213634491, 0.205352783203125, -0.4452736973762512, -0.4141717851161957, 1.6796029806137085, 0.23770877718925476, 1.5679435729980469, -1.2620506286621094, -1.0460749864578247, -0.38608357310295105, -0.30543243885040283, 2.005242347717285, 1.1625664234161377, 0.6157835721969604, 1.3757354021072388, -0.5845698118209839, 0.19595003128051758, 0.15250125527381897, -0.3216859698295593, -0.5251495838165283, 1.0082695484161377, 0.018182259052991867, -0.028862107545137405, 0.2134290635585785, 1.2442928552627563, 0.223025381565094, 0.4957161247730255, -1.6944046020507812, -0.5217873454093933, -0.18658433854579926, -0.1954277753829956, -1.2748225927352905, -0.03380519896745682, -1.0739898681640625, -1.0639915466308594, -0.5408577919006348, 0.7708869576454163, -0.5447354912757874, 0.33681705594062805, -0.20161844789981842, 1.0049673318862915, -0.24751190841197968, 0.26000043749809265, 0.9496336579322815, -0.5557637214660645, -0.8327792882919312, -2.10022234916687, -0.5718466639518738, -0.012780806981027126, -1.4900100231170654, -1.2709534168243408, 0.679990291595459, -0.9902591109275818, 1.8144391775131226, -2.1947884559631348, 1.0572689771652222, -0.6368069648742676, 0.5790994167327881, -0.03510995954275131, -0.14353783428668976, -1.6921075582504272, 0.13229551911354065, 0.14111341536045074, 0.277458131313324, 1.3127574920654297, -0.8951829075813293, 0.9614686965942383, 0.7534445524215698, -1.2451050281524658, -2.062809467315674, -1.4377448558807373, 0.05738350376486778, -0.7689782977104187, -1.8260242938995361, -0.7967139482498169, 0.4389902949333191, 0.7973684668540955, 0.7953158020973206, -1.3958559036254883, 0.21740350127220154, 0.5988420248031616, 0.848766028881073, -0.8719494938850403, -0.07197015732526779, 1.2642501592636108, -0.6917684078216553, -0.9135850667953491, 0.8146741390228271, 0.26252317428588867, 0.02917022444307804, 1.0046879053115845, -0.1627105474472046, -0.8186177015304565, 0.7289788722991943, 0.9089086055755615, -0.7901278138160706, 1.3517146110534668, -0.6752758622169495, -2.366176128387451, 0.05287463217973709, -0.32071617245674133, -0.6696258783340454, 0.9338557124137878, -0.8281172513961792], [1.1537131071090698, -0.8214334845542908, 0.06898725032806396, 0.6933797597885132, 0.448262095451355, -1.7299482822418213, 0.13762864470481873, 0.7294104099273682, -1.6284984350204468, -0.7256157994270325, 1.091052770614624, 0.2865392565727234, -0.17985492944717407, 1.1224521398544312, -1.0712755918502808, 1.1767151355743408, -0.622027575969696, 0.673751950263977, -0.06846880912780762, -0.5406374335289001, 1.2267588376998901, -2.519467353820801, -0.1818917989730835, 0.7985368371009827, -0.15369892120361328, 0.9588105082511902, -1.0868195295333862, 0.8548958897590637, 0.27978166937828064, 0.10981008410453796, -0.3730566203594208, 0.5706210732460022, 1.027368426322937, -0.5763455629348755, -0.5436042547225952, 0.23832197487354279, -0.2601872384548187, -0.445361852645874, -0.8559513092041016, 0.5379806160926819, -0.6268777251243591, -0.24823762476444244, -1.0939782857894897, -0.6076329350471497, 1.809512734413147, 0.6227036714553833, 0.6464993357658386, -0.20013666152954102, 1.75216805934906, -0.1769828349351883, 0.7658352851867676, 1.4140266180038452, 1.0224348306655884, 0.13357047736644745, 0.13514123857021332, 0.39225098490715027, -2.686119794845581, -0.43974295258522034, 2.0519564151763916, -0.26062098145484924, -0.09811794757843018, -0.711449384689331, -1.6468017101287842, -0.29321742057800293, 1.4285626411437988, 0.49557873606681824, 1.6826932430267334, 1.8356053829193115, 0.43078017234802246, 0.08932338654994965, -0.03637261316180229, 0.02042151428759098, -0.4157266318798065, 0.8391491174697876, 0.8686219453811646, 1.6019333600997925, 0.3236345946788788, 0.7556440830230713, 1.0364692211151123, 0.7695239782333374, 0.34338924288749695, -0.38470974564552307, 0.5354427695274353, -0.0054364511743187904, 0.11706189811229706, -1.1739991903305054, 0.283419132232666, -0.3237035274505615, -0.4002525806427002, 0.7519704699516296, -0.07385685294866562, -0.17563462257385254, -1.7664148807525635, -0.0005640905001200736, 0.7628870010375977, -1.6183923482894897, 0.16465218365192413, 1.0935261249542236, -1.1340421438217163, -2.9628634452819824, -0.2793869972229004, 0.04623159021139145, -1.122818946838379, 0.6981483101844788, 0.699845016002655, 1.138440489768982, 0.6740210056304932, 0.47798505425453186, -0.6899049282073975, -1.0793479681015015, -0.6024608016014099, 0.14050398766994476, -1.0342577695846558, 0.7822548151016235, 1.6511837244033813, 0.5091966390609741, 0.9564719796180725, -0.9003150463104248, -0.21306617558002472, -1.9092711210250854, -0.6111341714859009, -0.4941995441913605, -0.4033162295818329, -1.29049813747406, -1.5176790952682495, 1.4755504131317139, -1.7957780361175537, -1.0952702760696411], [-0.4409048855304718, -0.802818238735199, -0.07083680480718613, 0.09511970728635788, -2.4634206295013428, -0.7842642664909363, 1.0603991746902466, 1.337828516960144, 1.2433466911315918, -1.2837647199630737, 1.6227426528930664, 1.831504225730896, -0.19468234479427338, -0.8882285356521606, -0.7791088223457336, 1.9049619436264038, -0.23489248752593994, 0.02446192130446434, 1.1406184434890747, 2.6256861686706543, 0.10495420545339584, 0.7423610091209412, -0.22607015073299408, -0.3105258047580719, 0.9301168918609619, -0.5480623841285706, -0.10337946563959122, -0.6838639378547668, 1.00733482837677, 1.1180602312088013, 2.491546392440796, 1.1855207681655884, -3.0788888931274414, 0.2997075319290161, -0.8770543336868286, 0.7946668863296509, -0.5445777177810669, -1.5219519138336182, -0.70379638671875, 0.44763466715812683, 1.3770267963409424, 0.8974094986915588, 0.2585412859916687, 0.7236241102218628, -0.9508488178253174, 2.4116218090057373, -0.12303994596004486, 0.15382201969623566, 0.6383587718009949, -0.6828066110610962, -0.12299016863107681, 1.0802878141403198, -0.6828979849815369, -0.20913715660572052, -0.5828081965446472, -0.04638942331075668, -1.38335120677948, -1.4280409812927246, -0.2224251925945282, 0.4422013759613037, -2.142899751663208, -1.5510445833206177, -1.2718790769577026, 1.146885871887207, 1.3627811670303345, 1.494302749633789, -1.071700930595398, -1.7042745351791382, 1.2711172103881836, -1.1329782009124756, 3.0562584400177, 0.8443110585212708, 0.4495997130870819, 0.45460808277130127, -1.6980482339859009, 0.44318726658821106, -0.37167468667030334, 0.1631316840648651, 0.46878355741500854, 0.6695973873138428, 0.009381284937262535, -0.3785476088523865, -0.08118081092834473, -0.6501216292381287, 1.0751397609710693, 0.3405710756778717, -0.13613556325435638, 0.22249987721443176, 0.40689951181411743, 0.6158170700073242, -0.38826656341552734, -0.11839767545461655, -0.797713577747345, 0.06850734353065491, -1.1780654191970825, -0.2784372568130493, 0.9629256725311279, 0.7427963018417358, -0.4801499843597412, 0.9539660215377808, 0.3140435516834259, -0.09210389852523804, 0.7270525097846985, 0.440573126077652, 0.6451762914657593, 0.5469509363174438, 0.6990097165107727, -1.044190764427185, -0.6321289539337158, -0.9222623705863953, -1.8834733963012695, 0.32164499163627625, -0.09314499795436859, 2.264159917831421, -0.9958967566490173, 1.7000489234924316, -1.2203466892242432, -0.29607436060905457, -0.04713127017021179, -0.5520991683006287, 0.15771709382534027, 1.146485447883606, 0.36235037446022034, -0.4978421926498413, -0.10623997449874878, 0.18152859807014465, -0.13513413071632385, 1.3288427591323853], [-0.10484594851732254, -0.5848520994186401, 0.5457186102867126, -0.8033501505851746, 0.908845841884613, -1.0330654382705688, 0.5302083492279053, -0.203756183385849, 0.17368508875370026, -0.41460639238357544, -0.7032063007354736, -0.0347287654876709, 0.8042054772377014, 0.32750898599624634, 0.30309802293777466, 0.0884513407945633, -1.0492781400680542, -0.8291218876838684, -0.2675646245479584, 0.0861441120505333, 1.4375637769699097, -0.6310997009277344, -0.8229957818984985, 0.4032179117202759, -1.1628118753433228, -1.5584676265716553, -0.29009655117988586, 1.5292537212371826, 2.1050384044647217, -0.8551775217056274, 1.2118228673934937, -0.9586862921714783, 0.1816631406545639, -0.5249228477478027, -1.0529890060424805, 1.772233247756958, -1.7115148305892944, 1.0914993286132812, 0.63178950548172, 0.34353217482566833, 0.9509280323982239, 0.9796493649482727, 0.7239536643028259, -0.3848629295825958, 0.41539323329925537, 1.8592900037765503, -0.06720074266195297, -1.5791620016098022, -0.6104216575622559, -0.9443185925483704, -1.7213937044143677, 0.3472004234790802, -0.2724219262599945, 0.24057000875473022, 0.5556604862213135, 0.8386520147323608, 1.3633708953857422, 1.9177110195159912, -0.19967341423034668, -0.1526254266500473, -1.0133124589920044, -0.31516847014427185, -0.3570614159107208, -0.6204875707626343, -0.5842490792274475, -0.45853808522224426, 0.5329084396362305, -0.8891956806182861, -0.10808980464935303, -0.1995847225189209, -0.7119032740592957, -0.24364018440246582, -1.0410484075546265, -0.507243275642395, -0.038987141102552414, 0.431710422039032, 0.6877015233039856, -0.5859867334365845, 0.3089711368083954, -1.9925432205200195, -1.3058420419692993, -0.7863225340843201, 0.10991831868886948, 1.3389989137649536, 1.7216320037841797, 0.4420918822288513, 0.5105917453765869, 1.167235255241394, -0.3446590304374695, 1.7130937576293945, -0.22212615609169006, -1.3062809705734253, 0.24381446838378906, -0.4561930298805237, -1.6714965105056763, -1.197152853012085, -0.3191913068294525, -0.9587498307228088, -1.1825973987579346, 0.5734640955924988, -0.014049068093299866, 0.8029009103775024, -0.04347076639533043, 0.6479301452636719, 0.11372704803943634, 0.5750625729560852, -0.5987765789031982, -0.7839053869247437, -0.3763796091079712, 0.7103636264801025, 0.4203568696975708, -0.12413713335990906, 0.8840983510017395, 0.562468945980072, 0.4179477095603943, -0.47161078453063965, 1.9780397415161133, -0.6467087864875793, 1.0970368385314941, 1.460542917251587, 1.3274667263031006, 1.8506721258163452, 0.13906919956207275, 0.5607755780220032, -0.02330969274044037, -1.0242098569869995, -1.04104483127594, 1.059822916984558], [-1.2678630352020264, 0.6217015385627747, -0.5400785803794861, -1.4564152956008911, 1.6682401895523071, 1.3298449516296387, -0.17047028243541718, 2.950575351715088, -1.6510051488876343, -1.3767659664154053, 2.0161547660827637, -1.1083954572677612, -1.0073550939559937, -0.6334421038627625, -1.2770169973373413, -1.4027901887893677, 1.5181821584701538, -0.20750977098941803, -0.18683166801929474, -2.8832223415374756, -1.521519422531128, -1.2885373830795288, -0.8300071358680725, 0.15269190073013306, -0.8700466752052307, 0.5832368731498718, 0.8532834649085999, 1.6266086101531982, -0.479799747467041, -0.031322989612817764, 0.38159722089767456, 0.655659019947052, -0.5280676484107971, 0.6407076716423035, -0.7507140636444092, -0.9654656052589417, 0.09529805183410645, -0.6677600145339966, -0.5281347632408142, 1.049166202545166, 1.4578266143798828, -1.5840272903442383, -0.688804030418396, -0.22157752513885498, 0.4841049909591675, 0.38572561740875244, -0.40217533707618713, -0.8945457339286804, -0.37185147404670715, -1.4080275297164917, -1.6235945224761963, 0.05435623973608017, -0.0771767869591713, -0.45580312609672546, -0.6695565581321716, -1.821336030960083, 1.1759467124938965, 1.2152272462844849, -1.2245835065841675, -1.0456006526947021, 1.184942364692688, -0.38030409812927246, 0.14676660299301147, -0.7532799243927002, -0.5956568121910095, -0.5207935571670532, -0.4406448304653168, -0.9897651672363281, 1.8232669830322266, 0.9119613766670227, 1.8210991621017456, 0.2791837453842163, 1.5820549726486206, -1.2853158712387085, 0.5464402437210083, -0.5833737850189209, 0.07485183328390121, 1.4580082893371582, 1.6136901378631592, -0.3416074514389038, -0.2565461993217468, -0.8646761775016785, -0.44170260429382324, 1.5784326791763306, -0.3130544126033783, -0.7246617078781128, 1.0283267498016357, 0.2871563136577606, -1.0448731184005737, 2.182056188583374, 0.08967363834381104, -0.7906382083892822, 0.35043802857398987, -0.7153213024139404, -1.451269507408142, 0.9128398299217224, -0.4476446807384491, 0.7580687999725342, -0.8943117260932922, -0.26933789253234863, 0.7155206203460693, 0.0815860703587532, 0.014376489445567131, -0.6399487257003784, -0.9557005167007446, -1.4958748817443848, 1.3296297788619995, -0.5790818929672241, 0.35661789774894714, -0.08167485147714615, 1.8876582384109497, 0.6138163805007935, -1.5333325862884521, -0.32587969303131104, -0.6442775726318359, -1.3364571332931519, 0.10491408407688141, 0.47815313935279846, -0.3769875764846802, -1.2981081008911133, 0.6906461119651794, -0.055901240557432175, 1.838314175605774, 0.8228186964988708, 0.5144926905632019, 2.023055076599121, 0.4457109868526459, 1.4525456428527832], [-1.2631549835205078, -0.7290568947792053, 1.109479546546936, 0.28115448355674744, 0.7114424109458923, 0.268146276473999, -0.9324892163276672, 0.24901506304740906, 0.6658227443695068, 0.7451605200767517, 1.0171027183532715, -1.020504355430603, -2.2016942501068115, 1.0608105659484863, -1.6726243495941162, 2.2488582134246826, 0.23266805708408356, 0.6456114649772644, -0.18634308874607086, 0.03657511994242668, 0.8202539682388306, 0.3101288378238678, 0.5609868168830872, -0.8717619776725769, 0.4718817174434662, 0.16809414327144623, -0.36235326528549194, -0.7393373250961304, 0.19326288998126984, -1.5348128080368042, 0.22201761603355408, 0.07665517926216125, -1.497910976409912, 1.4104045629501343, -1.5668596029281616, 1.0105122327804565, -0.32141542434692383, -1.4774284362792969, -1.2009459733963013, -0.18164965510368347, 0.03167860582470894, 0.1759156733751297, 0.16426421701908112, -1.4431096315383911, -1.1364878416061401, 2.150275945663452, -0.015121678821742535, -2.1155381202697754, -0.44926637411117554, 0.44466596841812134, -0.48773330450057983, 1.552164077758789, -0.14180774986743927, 0.848683774471283, 0.41981634497642517, 0.8093540668487549, 0.3471565842628479, -0.19982154667377472, -0.7341407537460327, 0.11055009812116623, 0.17826931178569794, -0.45368435978889465, 0.3977830410003662, -2.1208231449127197, 0.1723557561635971, 0.7699755430221558, 0.7328338623046875, -0.39020949602127075, -0.14355918765068054, 0.29692474007606506, -0.5637650489807129, 0.9793065190315247, -0.16975577175617218, 0.09369809925556183, 1.0344773530960083, -1.0040631294250488, -2.2770888805389404, 0.580799400806427, -0.5422090291976929, 0.03314204514026642, -0.07766694575548172, 0.1593094766139984, -0.17231319844722748, -1.4690991640090942, 0.005572488531470299, 0.7725745439529419, 0.3638902008533478, -1.0541841983795166, -0.07645486295223236, 0.5959997177124023, 0.663210928440094, 1.617270827293396, -1.4265692234039307, -0.0209483802318573, 0.9441742897033691, 0.41338402032852173, 0.9609470963478088, 0.19349287450313568, 0.9610134959220886, 0.40530577301979065, -0.31331518292427063, -1.5384997129440308, 1.2138038873672485, -1.8453627824783325, 1.1509546041488647, -0.4950522184371948, 1.0027575492858887, -0.05854137986898422, -0.8330298662185669, 0.6019905805587769, 0.07326171547174454, -2.695246458053589, 0.9367987513542175, 0.09045889228582382, -0.3108390271663666, -0.6778140068054199, 0.18133874237537384, 0.07569149881601334, -0.44282838702201843, -0.22895564138889313, 0.2667347490787506, 0.19408603012561798, 1.0102325677871704, -0.7364428639411926, -0.1207922026515007, 0.22211681306362152, -1.5737693309783936, -1.3104909658432007], [0.008907350711524487, 0.001389614655636251, -1.3929463624954224, 2.3747494220733643, 1.8401519060134888, 0.5591205358505249, -1.2027629613876343, -0.3268950581550598, -0.3893776535987854, -0.0033864551223814487, 1.1649483442306519, -0.2951001822948456, 0.17721496522426605, -0.412166029214859, -0.6951320171356201, -0.40600892901420593, 0.32715529203414917, -0.06801891326904297, 0.5010573267936707, 1.151578664779663, 0.6407126784324646, -1.0056850910186768, 0.1489885002374649, -0.30898475646972656, 0.22427302598953247, 1.0947227478027344, -0.7532221674919128, 0.7383251190185547, -1.713887095451355, -0.4850936830043793, -1.5556433200836182, 0.1590839922428131, 1.7831528186798096, 1.317984938621521, 0.5862501859664917, 0.3820306658744812, 0.778914749622345, -0.22092317044734955, -0.404477596282959, -0.9695398211479187, 0.0014915455831214786, -0.22476829588413239, -1.3718715906143188, -1.5833806991577148, -0.10011056065559387, 0.6737661361694336, 0.24593843519687653, -0.8502050042152405, 1.3459093570709229, 0.4795570969581604, -2.7315053939819336, 0.8054850101470947, -1.6093617677688599, 1.5879764556884766, -0.49983784556388855, -1.2780413627624512, -0.154579296708107, -0.2156052589416504, -0.37768876552581787, 0.4430592954158783, 1.499009132385254, -0.799486517906189, 0.013947845436632633, -0.7868545651435852, 0.30095675587654114, 0.40289008617401123, 0.5931687951087952, 0.5962008237838745, 0.7292695045471191, 0.7585128545761108, -1.9123585224151611, 0.216949462890625, -0.34996655583381653, 0.6206392645835876, 0.36339741945266724, 1.330491542816162, 1.1735773086547852, 0.9830071330070496, 0.5779833197593689, -0.7419430613517761, -1.6198173761367798, -0.7616747617721558, 0.6537314653396606, 0.6208292841911316, 0.4590142071247101, 1.9043378829956055, -1.927088975906372, 1.2865595817565918, -0.22851777076721191, -1.7230451107025146, 0.37327510118484497, -0.19403570890426636, -0.8668729662895203, -0.8682513236999512, 0.8253288269042969, -0.17337538301944733, 0.4860597252845764, -0.7183646559715271, 0.10971364378929138, -0.48875993490219116, -1.4078627824783325, -0.15828581154346466, 0.36490654945373535, 0.8657721281051636, -1.266893982887268, -1.756813406944275, -0.6534087657928467, 0.9121459722518921, 0.07423454523086548, -0.7674898505210876, -0.47461143136024475, -1.9447708129882812, 0.29794400930404663, -1.1862068176269531, -0.32296615839004517, 0.32690995931625366, -0.16133859753608704, -0.08190622925758362, -0.2642751932144165, -0.31408652663230896, 2.1604137420654297, -1.5197571516036987, -0.11134491860866547, 0.10070446133613586, 1.10098397731781, -1.47666597366333, -1.8753471374511719, -1.2428890466690063], [0.7194961905479431, 0.31953907012939453, -1.2798162698745728, -1.5386570692062378, -0.22162175178527832, -0.5562637448310852, 0.5700831413269043, -0.22505201399326324, -0.9814375042915344, 1.392926812171936, -1.7545166015625, -1.2356703281402588, 1.3726260662078857, 1.1746084690093994, -2.2086679935455322, -0.49576741456985474, -0.6523581743240356, 0.726091206073761, 1.9745320081710815, -1.408289909362793, 1.1451491117477417, 0.5205678343772888, -1.4770524501800537, 0.007749434094876051, 0.5546969175338745, 1.1816356182098389, -0.5208981037139893, -0.6933255791664124, -0.16940519213676453, -0.45204588770866394, -0.34108489751815796, -1.138223648071289, -1.7012264728546143, -0.7849201560020447, -0.18818879127502441, -1.6370495557785034, -2.1748757362365723, 0.8269021511077881, 0.29915666580200195, 0.007358182221651077, -0.9420587420463562, 0.1106170266866684, -0.22421179711818695, -1.036242127418518, -1.180783987045288, -0.7767730355262756, -0.2530783712863922, 0.9635331630706787, -1.3609874248504639, -0.9227635264396667, 0.9936718344688416, 1.3860387802124023, 0.9046016335487366, 0.896952748298645, 0.47218945622444153, 0.33690622448921204, 0.7093927264213562, 0.26571813225746155, -0.07664217054843903, 2.3092117309570312, -0.8486781120300293, -0.7196037769317627, -0.39003366231918335, 0.06562858819961548, -0.20494964718818665, 0.8121962547302246, -1.6749544143676758, 0.02906598523259163, -0.4343198239803314, -0.3024621605873108, -0.29848605394363403, 2.2087862491607666, -0.3948782682418823, -0.17819486558437347, -0.041425641626119614, -2.0513217449188232, -0.9418870806694031, 0.08673076331615448, 0.6347516775131226, 0.477546364068985, -0.5274757742881775, -0.6397243142127991, -0.10228709131479263, 0.9466778039932251, 0.6954299807548523, -0.2176503986120224, 1.030984878540039, 0.9969092011451721, -0.6240046620368958, -1.5596067905426025, 1.4904297590255737, -1.1526530981063843, -0.3537188768386841, -0.27306562662124634, 0.2812182903289795, -1.2574089765548706, 1.9751311540603638, 0.059999242424964905, 0.45839551091194153, 0.49137070775032043, 0.5437594652175903, 0.9880373477935791, -0.07525163143873215, -1.4925473928451538, 2.0333967208862305, -0.966637372970581, 0.30486688017845154, 0.40911152958869934, 1.0250951051712036, -0.21014854311943054, -1.634710431098938, 0.038866590708494186, 1.1089140176773071, -0.48327523469924927, -0.5232459306716919, -0.26443254947662354, 0.7881928086280823, -0.5155506730079651, -0.6412414312362671, 1.0195837020874023, 0.5251656174659729, 1.5057719945907593, -2.2055206298828125, -0.5548801422119141, -0.9019600749015808, 0.0042623789049685, -0.7561963796615601, -0.13001592457294464], [-0.3144509792327881, -0.15611746907234192, -0.4786282479763031, -0.6513181924819946, 1.1505671739578247, -0.6656831502914429, 0.1287253201007843, 2.4050452709198, -1.509123682975769, -0.7111640572547913, 0.3959353566169739, -1.6043288707733154, -2.0980050563812256, -0.47883936762809753, 0.2393008917570114, -0.48422515392303467, 0.12838803231716156, -1.433367371559143, 2.3368585109710693, 0.6836337447166443, 0.04983071610331535, -0.2815100848674774, -1.4494267702102661, -0.6697478294372559, 0.2926102876663208, 0.6778362393379211, 0.6427303552627563, -1.0593986511230469, 0.411667138338089, -0.7382797002792358, -0.5839690566062927, -0.5206815004348755, 0.7902405858039856, -0.6104714274406433, -0.6558908224105835, -0.6871280670166016, -0.8755270838737488, 0.006923696491867304, -0.8118069171905518, -1.5280076265335083, -0.10230569541454315, 0.31990301609039307, 0.26720476150512695, 0.09740635752677917, 0.033455826342105865, 0.25734570622444153, 1.501994013786316, -1.8224053382873535, -0.6856404542922974, 0.5466456413269043, 0.23919238150119781, -0.8020320534706116, -0.2517295181751251, 0.9578768014907837, 0.07165659219026566, 0.8290839791297913, -2.024066925048828, -0.5261850953102112, 2.7579452991485596, 1.2548234462738037, -0.9515603184700012, -0.11122041195631027, 0.26344895362854004, -1.360045075416565, -0.49959537386894226, 0.527316689491272, -1.0872251987457275, -0.4423739016056061, -0.3164981007575989, 0.14364093542099, -0.5641413927078247, 0.23939287662506104, 0.099196657538414, 0.13331197202205658, -0.9984947443008423, 0.07723333686590195, 1.1991583108901978, -0.4026191234588623, 1.375108003616333, -2.481049060821533, 0.7397465109825134, 0.2741962671279907, -0.13649624586105347, 0.44762715697288513, -0.3897719979286194, -0.000516410858836025, 0.6209524273872375, -1.5667147636413574, 0.570853054523468, -0.27055269479751587, 0.7224644422531128, -0.03173994645476341, -0.10706104338169098, -0.15390805900096893, 0.8828096985816956, -1.220197319984436, 0.40074801445007324, 1.3908969163894653, 0.9937834739685059, -0.6837447285652161, -0.273710697889328, -0.2787191569805145, -1.2177138328552246, -0.05264350771903992, 2.1111319065093994, 1.1625458002090454, 0.33644187450408936, -0.23374389111995697, -0.8289977312088013, 0.13047920167446136, 1.627088189125061, -0.11706526577472687, -1.2839105129241943, -1.2676523923873901, -0.3705625534057617, -0.2220739871263504, 2.0443902015686035, -1.7624186277389526, 1.2279518842697144, 0.08120839297771454, 0.9350545406341553, 0.044942859560251236, -0.7900651693344116, -0.8045899271965027, -0.8401784896850586, 2.2750184535980225, 1.1784108877182007, -1.9450033903121948], [0.32931917905807495, 1.9542983770370483, 1.9606257677078247, 1.7840509414672852, -0.758618175983429, -0.6231887936592102, 0.9147258400917053, 1.04344642162323, -1.583311676979065, -0.8068037629127502, 0.6337066888809204, -1.1379600763320923, -1.79814875125885, 0.4331919252872467, -0.35579532384872437, -0.30136460065841675, -2.127206802368164, 0.475726455450058, -0.03387666493654251, 2.003221273422241, 0.4642221927642822, 1.181225061416626, -0.13361269235610962, 0.8029252290725708, -0.12318204343318939, -2.039703845977783, -0.4212327301502228, -1.0938730239868164, 0.25117456912994385, 1.9194327592849731, 0.35672685503959656, -0.5374223589897156, 1.1471620798110962, -0.9319814443588257, 1.4398092031478882, -0.2955101430416107, -1.0493322610855103, -0.6831772923469543, -0.3587028384208679, -0.3063622713088989, -1.547258973121643, -0.17237715423107147, -0.10566477477550507, 2.411165952682495, -1.9827497005462646, 1.1871790885925293, -2.8051538467407227, -0.8327367305755615, -1.1329379081726074, 0.41249382495880127, -0.22621306777000427, 0.5559682250022888, -1.1040780544281006, 0.5196835994720459, 0.5674611926078796, 0.6931515336036682, 0.3146694302558899, -0.9552320837974548, 1.0118457078933716, 2.0737364292144775, 0.9337905645370483, 0.48564234375953674, -1.0282057523727417, 0.6884482502937317, 2.046705722808838, 0.6673883199691772, -0.8895907998085022, 0.0648234561085701, 1.5433101654052734, 0.0061042229644954205, -0.7590191960334778, 0.1461777240037918, 1.1287200450897217, 1.222846508026123, 0.36582151055336, -1.9396803379058838, 0.3398553729057312, -0.7528315782546997, -0.18848508596420288, 0.21933820843696594, 0.46838587522506714, 0.4199278950691223, 1.975728154182434, 1.2790625095367432, 0.20150339603424072, -1.7732617855072021, -1.0943853855133057, 0.365464985370636, 1.157934308052063, -0.2872801721096039, 0.7698266506195068, 0.42654022574424744, 0.007796918042004108, 1.6341009140014648, 0.9396937489509583, -0.4386811852455139, 0.25363221764564514, 0.43085935711860657, 0.33281999826431274, -0.750879168510437, 0.8674307465553284, -0.1654079705476761, 0.5017716884613037, -0.7229197025299072, -1.2544217109680176, 0.1021009013056755, -0.018541933968663216, -1.0175087451934814, 1.0784088373184204, 0.07652289420366287, 0.5654518604278564, -0.7641130685806274, 0.7461007237434387, -0.012329674325883389, 1.9596010446548462, 0.031915564090013504, -1.9870188236236572, 0.24086681008338928, 0.5600166320800781, 1.0174064636230469, -1.1981416940689087, -3.221674680709839, -0.18877071142196655, 0.35958510637283325, 0.3768942952156067, -0.7766764163970947, -3.6211509704589844, -0.3970705568790436], [-1.7843095064163208, 0.02716774493455887, -0.8300690650939941, -0.3426949679851532, 1.314971923828125, -1.152303695678711, 0.1287381649017334, -0.43803462386131287, -2.413268804550171, 0.3298686742782593, -1.4248807430267334, 1.2641621828079224, 0.49852660298347473, -0.5192607641220093, -0.4265824556350708, -0.3944152891635895, 0.07207644730806351, 0.3503348231315613, -0.5006691217422485, 0.2491789609193802, 0.9052947163581848, 0.5806366801261902, 0.6741019487380981, 2.684654712677002, -0.01988319307565689, -1.279380202293396, -0.6936812400817871, 0.20188945531845093, 0.36345162987709045, 0.9018676280975342, -0.672625720500946, 0.5869889855384827, 0.1986045390367508, 1.0755741596221924, -0.6434378027915955, 0.43693774938583374, 1.877028226852417, -0.4380623400211334, 0.6254349946975708, -0.8335393071174622, 0.33719176054000854, 0.7879917621612549, 0.4680764377117157, -0.9780993461608887, 0.40487149357795715, -0.07690445333719254, 1.2502270936965942, 1.6306805610656738, 0.4379188120365143, 0.8418063521385193, 0.777419924736023, -0.37774768471717834, -0.6208637356758118, -0.40092766284942627, 1.0469756126403809, 1.1566367149353027, -1.9371486902236938, 0.44710543751716614, -0.3925018012523651, 0.8916487097740173, 1.2026357650756836, 0.20326785743236542, -1.0074046850204468, 0.24596385657787323, 1.0087379217147827, 0.6351093649864197, -0.7489380836486816, 1.006456732749939, -0.5788404941558838, -1.9128191471099854, -1.151976227760315, 0.906060516834259, -0.19479788839817047, -0.5970464944839478, 0.004658891353756189, 1.010036587715149, 0.3852638602256775, 1.4835158586502075, -1.4977056980133057, -0.6068407297134399, 1.0309793949127197, -0.32125478982925415, -0.8033186793327332, 0.12332162261009216, -0.0653616264462471, -0.8624488711357117, 0.6215085983276367, -0.23732398450374603, 0.6109269857406616, -0.1387718915939331, 0.08311927318572998, -0.7400631308555603, 1.978318452835083, 1.041958212852478, 2.854783058166504, 0.1265585720539093, 0.7751715183258057, 0.4663660526275635, 0.8895746469497681, -0.9157994985580444, 2.1730446815490723, -0.6858832240104675, -0.6182947158813477, -1.163622498512268, 0.13291068375110626, 1.766481876373291, 0.28826168179512024, 0.1331554800271988, 1.2862142324447632, -1.2410491704940796, -0.46576109528541565, 0.09758084267377853, 0.34848037362098694, -1.2217323780059814, -0.32598626613616943, -1.4024580717086792, -0.29202353954315186, -0.9436408877372742, -1.3596010208129883, 1.187620997428894, 1.4767731428146362, -0.5495695471763611, -0.3102111220359802, -0.12614712119102478, 0.45497363805770874, -0.8880414962768555, -0.09086278080940247, 0.7396097183227539], [0.32701775431632996, -0.26909497380256653, -0.944037139415741, 0.7130690217018127, 0.6752275228500366, -0.07238710671663284, 0.7599326968193054, 0.7009103894233704, 0.32986345887184143, -0.09129592031240463, -0.7432600855827332, -0.15758271515369415, 0.9840576648712158, 2.1774582862854004, -0.5001078844070435, -0.2972354590892792, -0.9115398526191711, 0.7936503887176514, -0.7365768551826477, -1.2322778701782227, 0.749646782875061, 1.6278221607208252, -0.6035726070404053, -0.14620870351791382, 0.10162460803985596, -0.22900615632534027, 0.28516149520874023, -1.9211689233779907, -1.1862949132919312, -1.171076774597168, -0.19488486647605896, -0.896477460861206, -1.0481475591659546, 0.733799159526825, -0.8177309632301331, 0.8858560919761658, -0.9582267999649048, -1.6057039499282837, 0.3358737826347351, -0.2907602787017822, 2.6449081897735596, 0.25074270367622375, 2.674933433532715, -0.25845932960510254, -0.09044792503118515, 2.7323296070098877, 1.368186354637146, -1.3385781049728394, 1.4915461540222168, -1.965989351272583, 0.6697372794151306, -1.0338151454925537, -1.7714649438858032, 0.6268035769462585, 2.5002458095550537, -0.3006785213947296, 0.5979146361351013, -0.5461506247520447, 0.024341117590665817, -0.7263811826705933, -0.5125162601470947, 1.1865081787109375, 0.22640031576156616, -0.8111147880554199, 0.615929126739502, 0.4045563042163849, 0.9088910222053528, -0.37155547738075256, 2.0438971519470215, 0.14835496246814728, -1.959765076637268, -1.1816328763961792, -0.36282339692115784, -0.65468829870224, 0.7397582530975342, -0.036219753324985504, 0.574531078338623, -1.1067767143249512, -0.7495592832565308, 0.9903547763824463, 0.5518047213554382, -1.1831713914871216, -0.017490018159151077, -0.8745895028114319, -1.2257781028747559, 0.4566520154476166, -1.1098028421401978, 0.6468386650085449, 1.7511513233184814, 1.3499139547348022, 0.6364076733589172, 0.2878701984882355, 0.057392556220293045, 2.4459216594696045, 0.39837461709976196, 0.7233695387840271, 2.6825766563415527, -0.4870585799217224, 2.3319602012634277, -2.050835132598877, 1.1469639539718628, -0.7512837648391724, -1.2289901971817017, 1.6615132093429565, -0.4111827611923218, -0.7990343570709229, 0.2656899392604828, 1.1986522674560547, 0.5440614819526672, -0.8446000218391418, 0.6553343534469604, 1.5163356065750122, 0.2743329107761383, 0.9487909078598022, -0.5617547631263733, 0.3205004334449768, 0.6234040260314941, -1.5560599565505981, 0.4691632091999054, 1.016320824623108, -0.001018167706206441, 0.25101083517074585, -1.9755522012710571, 1.7419676780700684, -0.33104097843170166, 0.6173534989356995, -0.8273842334747314, -0.6639267206192017], [0.7019575834274292, -1.6477410793304443, -0.629765510559082, 0.11853467673063278, 0.344313383102417, 0.26038986444473267, 0.36300480365753174, 0.7704533338546753, 1.7081700563430786, -0.14863276481628418, -1.6173073053359985, -0.6753908395767212, -0.7880932688713074, 1.0016041994094849, -0.39088335633277893, -0.46874743700027466, -2.0093021392822266, -0.2984072268009186, 0.2924150228500366, -0.0636005848646164, 0.9272989630699158, -0.5014864802360535, -0.5485890507698059, -1.8182207345962524, 0.8271161913871765, -1.3156615495681763, -1.1388099193572998, -0.19719651341438293, -0.8591446876525879, -1.2672704458236694, -0.8876858353614807, 0.19425809383392334, -0.9172865152359009, 0.8926337361335754, 0.33694103360176086, -1.079779028892517, -1.7107120752334595, -0.616858184337616, 0.07344871759414673, 0.3152521848678589, 0.17109082639217377, -1.7161201238632202, 0.6779198050498962, 0.2086045742034912, -0.15613673627376556, -0.19281232357025146, -0.7545937299728394, 0.04077441245317459, 1.589816927909851, -0.3197034001350403, -0.21937398612499237, 0.3882751166820526, -0.31906238198280334, 0.9127013683319092, 1.439117431640625, -0.26611796021461487, -1.004560112953186, 0.33266767859458923, 2.5041420459747314, 0.8787223100662231, -1.0514650344848633, 0.027019549161195755, -0.4385063648223877, -0.5443044304847717, -0.5735507011413574, 0.8760265111923218, 1.9105157852172852, -0.6542444825172424, -0.6756240725517273, -0.3945128619670868, 1.421706199645996, 0.43965309858322144, 0.4965650737285614, 1.470096468925476, 0.12265461683273315, 0.8552051186561584, 0.455230176448822, 0.12601202726364136, -1.0029913187026978, 0.4065692126750946, 0.2941070795059204, -0.4517955183982849, -0.8286258578300476, 0.08847105503082275, 0.28299885988235474, -0.9391708970069885, -0.463186115026474, 0.021930227056145668, -1.6649442911148071, -0.6824023127555847, 1.1233553886413574, 0.46961531043052673, 0.2970540225505829, 0.1330859214067459, 1.1537725925445557, -1.7277239561080933, -4.363180637359619, 0.7762138843536377, 1.1679248809814453, -1.1498661041259766, -0.3513847887516022, -0.3818649351596832, 1.9695305824279785, -0.5861383080482483, -0.3523280918598175, -1.3454480171203613, 0.37217584252357483, -0.7337340712547302, 1.3568129539489746, 0.5165210366249084, -1.4728260040283203, 1.399782657623291, -1.2608498334884644, -0.6936841011047363, 2.1965816020965576, 0.2605055570602417, -0.05104450508952141, 0.011770006269216537, -2.4353559017181396, 0.22106710076332092, -0.34128203988075256, 0.18727609515190125, 2.5104238986968994, 0.9386724829673767, 1.9511295557022095, -0.9544048309326172, 0.9150396585464478, -0.5757725238800049], [-1.1473225355148315, -0.9700369834899902, 1.322807788848877, 0.9407771825790405, 0.38245439529418945, 0.03696088492870331, 1.666276216506958, 0.13099175691604614, -0.21345698833465576, -0.2774946391582489, -0.9772651195526123, -0.05021796002984047, -0.7951170206069946, -1.697959303855896, 0.8848859071731567, -0.2581283748149872, -0.7475817799568176, 0.5724964141845703, 0.5553454160690308, -0.25553077459335327, 0.26486894488334656, -0.7128177285194397, 0.9192713499069214, 0.6148650646209717, 2.020744800567627, -0.9796350598335266, -1.3017901182174683, 1.4855761528015137, 2.336380958557129, 1.2068241834640503, -0.7073491811752319, -0.018063370138406754, -1.5078978538513184, -1.200910210609436, -0.9941104054450989, 1.0411914587020874, 1.4400320053100586, -0.4054880738258362, -0.05329994857311249, 1.2199859619140625, -0.09116343408823013, -1.3185523748397827, -0.17243075370788574, -2.0423665046691895, 0.5697525143623352, -1.7290418148040771, -0.947207510471344, -0.5666529536247253, 0.018449952825903893, 0.14692942798137665, 0.8569585084915161, -1.2687146663665771, -0.215373694896698, -0.3872678279876709, 1.80897057056427, 1.7263472080230713, -0.3834628760814667, -1.049532175064087, -0.03187020868062973, 0.9560542702674866, 1.4158953428268433, -1.495805263519287, -0.27788427472114563, -0.44576239585876465, 0.161427840590477, 1.5851497650146484, -0.16153714060783386, -0.08670248091220856, -1.2214539051055908, -1.169791579246521, -0.9790870547294617, -1.1451400518417358, -1.2123788595199585, -2.7976162433624268, 1.1818900108337402, -1.2371288537979126, 0.11325449496507645, 0.00836271047592163, -0.8406289219856262, -0.4983906149864197, -1.5388195514678955, -0.4971727132797241, -0.32625630497932434, 1.7490633726119995, 0.7983509302139282, 0.6453620791435242, 0.3933796286582947, 0.31251999735832214, 0.5456317663192749, 0.4763621389865875, -0.457207053899765, -1.8497352600097656, 0.3026266396045685, -0.2652723789215088, -0.829047441482544, 0.4230315089225769, -0.7864193916320801, -1.136102557182312, -1.1882010698318481, 1.1108382940292358, 1.3939578533172607, -1.3389707803726196, -1.9402127265930176, -0.7001751661300659, 2.476778984069824, -0.2690173387527466, 1.3537153005599976, -0.36357229948043823, -1.9478309154510498, -1.512387990951538, 1.4396666288375854, 1.2828385829925537, 1.0301344394683838, 1.5245953798294067, 0.36805614829063416, -1.7784905433654785, -2.1998167037963867, 0.1762360781431198, 0.5119538903236389, 0.6924895644187927, -0.7233914136886597, -0.688367486000061, -0.876723051071167, -0.9018690586090088, -0.1906796544790268, -0.19342753291130066, -1.47585928440094, 0.11714322119951248], [0.3011878728866577, -0.33360904455184937, 1.08119797706604, -0.9442914128303528, 1.0695685148239136, -1.1879514455795288, -0.9988773465156555, 0.05187511816620827, -0.041899487376213074, 0.3221694231033325, 0.21385551989078522, 0.9483530521392822, -0.5630037784576416, 0.7518431544303894, -0.616260290145874, 0.36613941192626953, 0.8374307155609131, 0.6347436904907227, -1.2112361192703247, 1.3535407781600952, -0.752215564250946, -2.4053468704223633, 1.2446612119674683, 1.60148286819458, 0.32146865129470825, 0.2822697162628174, 0.5058732628822327, -0.44992172718048096, 1.1147702932357788, -0.8613519668579102, -0.13822895288467407, -0.540031909942627, -1.2935304641723633, -0.35182517766952515, 0.29677918553352356, 0.9191951155662537, 0.5619639754295349, 0.8680722713470459, -0.012720638886094093, 0.07699166238307953, 0.5265111923217773, 1.0358099937438965, -0.6750857830047607, 1.092695951461792, -1.9318445920944214, -0.9264407157897949, 0.36206817626953125, -0.9474324584007263, -1.3058462142944336, -1.500407099723816, -1.4714603424072266, 0.3688567280769348, -1.387686014175415, 0.6732981204986572, 0.2744807302951813, 0.1772954761981964, -1.7309640645980835, -0.27555516362190247, 0.8145590424537659, 1.2326680421829224, -0.6762927770614624, -1.9093983173370361, -1.2042295932769775, 0.13261622190475464, -0.11443261057138443, -0.581925630569458, 1.184373378753662, -1.0037884712219238, 0.9160827994346619, -0.8764712810516357, -0.5355451107025146, -1.1701191663742065, -1.482689619064331, -0.16386516392230988, 1.5089175701141357, 0.06337329000234604, 0.2060166746377945, 1.3265423774719238, -2.643578052520752, 0.005131034180521965, -1.3870185613632202, -0.9578636884689331, 0.12859506905078888, 0.578380823135376, -0.3399483859539032, 1.0072664022445679, 1.0209640264511108, -0.5801589488983154, -0.21816104650497437, 0.18263478577136993, 2.0449843406677246, -1.5100189447402954, -0.02528277412056923, 1.2881150245666504, -0.023790651932358742, -0.8336032032966614, 0.5422899127006531, 0.6443725824356079, -0.09908830374479294, -0.21472515165805817, -1.0407373905181885, -1.6962655782699585, -0.2937244176864624, -0.43569400906562805, -0.14710400998592377, -0.3952232897281647, 0.46528923511505127, -0.5764231085777283, -1.1528325080871582, 0.22778525948524475, 0.14343078434467316, 1.277442216873169, 2.315412759780884, 0.21699652075767517, -0.6939254403114319, 0.07977060973644257, -1.4837369918823242, -0.1510985642671585, -0.00751509191468358, 1.241200566291809, -0.72365802526474, -0.16661188006401062, 0.14191216230392456, 1.1785814762115479, -0.053844671696424484, 1.099743366241455, -0.865973174571991, 0.9007774591445923], [0.9721095561981201, -0.08810615539550781, -1.1271693706512451, -0.051162127405405045, 1.368597149848938, -0.8742876052856445, -1.3012595176696777, 0.9231998324394226, -0.5732713937759399, -0.6425166130065918, 0.8345516920089722, 0.36450615525245667, -0.44302308559417725, -0.836784839630127, -0.5623120069503784, 0.7332783937454224, 0.6322163343429565, -1.3119819164276123, -2.4966981410980225, -0.19284707307815552, 0.35530927777290344, -1.3764535188674927, 0.632851779460907, 1.5253944396972656, -0.9661116600036621, -0.20473800599575043, -2.4314074516296387, 0.302024781703949, 0.07773459702730179, -0.3441227376461029, 0.6403009295463562, 0.5578630566596985, 0.7208625674247742, 1.3624906539916992, -0.027453213930130005, 0.36359694600105286, 0.07249011844396591, -0.027094103395938873, -0.6949378848075867, 0.3882407546043396, 0.32365384697914124, 0.7483614683151245, -0.39329805970191956, 0.005547969602048397, 1.9732707738876343, -0.751122236251831, -0.7709919214248657, -0.24705854058265686, -1.138005018234253, 0.7747295498847961, 0.4647713601589203, -0.47363001108169556, -1.8875445127487183, -0.7000211477279663, -0.2330876886844635, -1.2787678241729736, 0.9808782935142517, 1.6734354496002197, 0.3792071044445038, -0.10003337264060974, -0.9738906025886536, 0.9888956546783447, 0.6400623917579651, 0.020820893347263336, -1.3120087385177612, -1.3714743852615356, 0.6437838673591614, 0.409626305103302, -0.7501609325408936, 0.04561576992273331, 0.2732195258140564, -0.1318688690662384, 0.34117209911346436, -0.16989666223526, 0.2884717285633087, -0.6466618776321411, 0.18960702419281006, -0.11265356093645096, 0.6787514686584473, -1.3811461925506592, -1.2196509838104248, 1.919972538948059, 0.2833147346973419, -1.0114792585372925, 0.4145660698413849, 0.6383192539215088, -0.9856671094894409, -0.10603199899196625, 0.186941996216774, 0.2864399254322052, 0.5112936496734619, -0.14107699692249298, -2.24186110496521, 0.1404803842306137, 0.9000908136367798, 0.2843363881111145, -0.265092670917511, 0.13296741247177124, 0.5397732853889465, -0.07694122940301895, 0.16509640216827393, 1.6902600526809692, -2.8718669414520264, -0.8429751396179199, -0.903203547000885, -0.8289703726768494, 1.025881052017212, -1.5584242343902588, -0.5519375205039978, 0.42886289954185486, -0.9727510213851929, 0.3669168949127197, -1.43017578125, 2.61574649810791, 0.14801929891109467, -0.8312296867370605, -2.4354350566864014, 1.693277359008789, 0.43633198738098145, -1.215092658996582, -0.6630201935768127, 0.4403623342514038, 1.5019068717956543, -0.47686484456062317, 0.5582101941108704, -0.5712785124778748, 0.30513015389442444, 0.14362937211990356], [-0.4112248122692108, 0.24814994633197784, 1.4548407793045044, 1.6971231698989868, 0.6679256558418274, 1.0155504941940308, 0.31531378626823425, -0.24966804683208466, -0.5831143260002136, 0.9399705529212952, -0.8851906061172485, -0.5092325210571289, 1.8363271951675415, 0.06205674633383751, 0.15586915612220764, -0.9531523585319519, -2.675811529159546, -0.055973879992961884, 1.0011351108551025, -0.5969932079315186, 2.3059632778167725, -0.06674331426620483, -0.052095621824264526, -1.0856863260269165, -0.21938028931617737, 1.544010877609253, 0.040614306926727295, 0.041437067091464996, -0.07621201872825623, 1.1013779640197754, 0.02822120673954487, 0.7680954933166504, 1.3148958683013916, -0.18723735213279724, -1.6241672039031982, -0.10666017234325409, -0.7402527332305908, 1.0620167255401611, -0.032534558326005936, 0.4218108355998993, -1.3433144092559814, -0.9484469294548035, 1.2259272336959839, -2.113104820251465, 1.1758699417114258, 1.2983546257019043, 1.3480373620986938, 1.8297873735427856, 1.7359882593154907, -2.084437131881714, 0.19156521558761597, 0.9283189177513123, -0.5895222425460815, 0.1834528148174286, -0.6125339269638062, 1.4637147188186646, 0.4874032735824585, 0.430118590593338, -1.6893850564956665, 1.0286498069763184, -0.07526250928640366, 1.0845203399658203, -0.6121665239334106, -1.4947917461395264, -0.46672043204307556, 0.4602690041065216, -0.05471724271774292, 1.1198034286499023, 0.13748091459274292, 0.8978204131126404, -0.1835738867521286, -3.143779993057251, 0.6281360983848572, -0.6061047911643982, -1.1832233667373657, 0.30608445405960083, 0.3384961783885956, 0.045024141669273376, -0.01427990198135376, -0.7590611577033997, 0.17120301723480225, 2.0206966400146484, -0.8793234825134277, -0.6596152782440186, 0.8350315690040588, 0.6480082273483276, -0.7514789700508118, 0.05825768783688545, -0.6057755351066589, 0.6054394841194153, 0.8369665145874023, 1.6614147424697876, 0.8832195401191711, -0.2298564463853836, 0.12138453125953674, -0.5118896961212158, 0.03942979499697685, -0.2288762778043747, 0.5421682000160217, 0.8065458536148071, -0.8525224328041077, -0.9609179496765137, 1.7702451944351196, 0.6782214641571045, -0.05155254155397415, 0.0335577055811882, 0.7121061086654663, 0.7109271883964539, 1.6941207647323608, -1.5681477785110474, 0.10885678231716156, 1.3516755104064941, -1.3529300689697266, 1.1377732753753662, -0.2818397283554077, -1.1687259674072266, -1.322255253791809, -0.6525996327400208, 0.49060338735580444, 0.448045015335083, -0.3877982497215271, 1.027126431465149, 0.8595350980758667, 1.3352630138397217, -0.348177045583725, -0.6761065125465393, -0.10321822762489319, 0.2950890064239502], [1.3172411918640137, 1.0143866539001465, -1.6018993854522705, 0.8116103410720825, 0.19945183396339417, -0.2842656672000885, 1.4250596761703491, 0.4133400321006775, -0.14525535702705383, -0.07161968946456909, 1.104340672492981, -0.016549307852983475, -0.8359427452087402, 0.6041370630264282, -2.21978759765625, -1.5129343271255493, 1.1820061206817627, 0.505527138710022, -0.5835411548614502, -0.6515536308288574, 0.7799631357192993, -0.5358152389526367, -0.8261777758598328, 0.36681291460990906, -0.6346651315689087, -1.1112114191055298, 0.8130343556404114, -2.1872799396514893, -0.3767746090888977, -0.7662186026573181, -0.21793891489505768, -0.2822437286376953, 1.384151816368103, -0.7719086408615112, -1.8666960000991821, -0.78849858045578, -0.7609601020812988, 0.41012969613075256, 0.2628743350505829, -0.2627599537372589, 0.825769305229187, -2.3263113498687744, 0.15892380475997925, -0.12399177253246307, -0.4561958312988281, -0.9378649592399597, -1.252711296081543, -0.9414933919906616, -0.967195987701416, -1.5398499965667725, -1.7791939973831177, 1.4691258668899536, 1.5106542110443115, -0.5427283048629761, -0.6860626935958862, -0.7375332713127136, 0.4099700152873993, 0.08047256618738174, 0.6212692260742188, 0.5099040269851685, 0.7045825123786926, 0.07769768685102463, -0.6495197415351868, -0.782507061958313, 1.8782762289047241, 0.018131891265511513, 0.8125710487365723, -0.7184548377990723, 0.4351969063282013, -0.8362804651260376, 1.4775546789169312, 0.6456533074378967, -1.3094240427017212, 0.3332495391368866, 0.48307791352272034, -1.8228366374969482, 0.6491621136665344, 0.9366341829299927, 2.464787721633911, -0.08402512967586517, 1.9743759632110596, 1.5183610916137695, -0.11463157832622528, 0.9433620572090149, 1.0370393991470337, -0.37213313579559326, 2.369236707687378, 0.35750705003738403, 0.7965986728668213, 0.08321450650691986, -0.6467335820198059, 0.09072758257389069, 0.702709436416626, 2.4643447399139404, -1.14934241771698, -0.478176087141037, -1.4976871013641357, -0.9453938007354736, -0.6173127293586731, 0.19955068826675415, 0.5063514113426208, -0.31436964869499207, -0.7763786911964417, 1.3539623022079468, -0.38424816727638245, -0.7941713929176331, 0.8588294386863708, -0.39658331871032715, -0.25982484221458435, -0.5176734924316406, -0.17735105752944946, -0.349045068025589, 1.492091178894043, 0.10245489329099655, 1.3177629709243774, -0.4369503855705261, -1.097631573677063, 0.4838120639324188, 0.4817488491535187, -1.3554794788360596, -0.6020026803016663, -0.6974664926528931, 1.2700042724609375, 0.09775584936141968, -0.29592180252075195, 1.9389052391052246, -0.9771836400032043, -2.402766704559326], [-0.3301922082901001, 1.9733892679214478, -0.5015354752540588, -1.0367071628570557, 0.15913647413253784, -1.171155571937561, -0.21828030049800873, 1.0428110361099243, -1.3719314336776733, 0.4243835210800171, 0.5121450424194336, -1.2486580610275269, 1.0812673568725586, 0.7604484558105469, 0.38569486141204834, 0.11925598233938217, 0.015008443966507912, -1.8996652364730835, -1.9013795852661133, -1.338965892791748, -0.119136281311512, -1.3557718992233276, 0.6531410217285156, -1.011011004447937, 1.8478530645370483, -0.8905607461929321, 0.5317725539207458, -0.5364161133766174, -0.29204419255256653, 0.13898026943206787, 0.5160443782806396, 3.9564595222473145, 1.8202687501907349, -0.1510099470615387, -0.6740396618843079, -0.8129040002822876, 0.7506586313247681, 1.5103609561920166, 0.9085155725479126, 0.05458398908376694, 0.041873447597026825, -0.21178559958934784, -0.19054806232452393, 1.484082818031311, 1.0999157428741455, 1.8661895990371704, -1.9476438760757446, 0.2732096314430237, 0.179609015583992, -0.11035309731960297, 0.3429739773273468, -0.02488783933222294, -0.8961191177368164, 0.15986444056034088, 1.3390893936157227, -0.7824077606201172, -0.7013934254646301, 0.9631325006484985, 0.5735061764717102, 2.8503236535470933e-05, -0.537010133266449, 0.10524602234363556, 0.5748583078384399, 0.6229844689369202, -0.5049483776092529, 0.02719581499695778, 0.37217289209365845, -1.471780776977539, -0.6989904046058655, -0.36819955706596375, -0.08155057579278946, -0.11283788830041885, 0.30589762330055237, 0.19865627586841583, -1.0919219255447388, 0.2451746016740799, 1.5637855529785156, -1.1625640392303467, 1.8765751123428345, -0.1585962027311325, 0.39726999402046204, -1.056391716003418, 0.35443395376205444, -0.018748179078102112, -1.023112416267395, -1.734043836593628, -0.3381383419036865, 1.3076809644699097, -0.7521389722824097, 0.06114910915493965, -0.7484266757965088, -0.023225076496601105, 1.1664315462112427, -0.488707959651947, -0.8037422895431519, 0.46860989928245544, 0.9928614497184753, -0.21687756478786469, 1.2008813619613647, 0.3572271168231964, 0.12907999753952026, -0.9228337407112122, 0.5605263710021973, 0.10318867862224579, 0.67204350233078, -1.563900351524353, 0.6322258710861206, 0.2935534119606018, -0.7552167773246765, -0.10930344462394714, 0.5462968945503235, -0.9859604835510254, -1.7435812950134277, -0.22449731826782227, -0.22552092373371124, -0.6316737532615662, -0.3395349085330963, -0.204965740442276, 0.7397465705871582, 1.6356220245361328, 0.17025820910930634, -0.5422412157058716, -0.4981236457824707, 0.7248103618621826, 1.021116018295288, -1.2169290781021118, 2.789095640182495, -0.3143382668495178]], [[0.19789081811904907, 0.5879193544387817, -1.3184376955032349, 0.9903567433357239, 0.8177779316902161, 0.7743068337440491, -0.616085410118103, 0.24398937821388245, 0.6814699769020081, -0.8360717296600342, -0.8079847097396851, -0.24911117553710938, -2.584110736846924, -0.35607394576072693, 1.3370469808578491, 0.005968076176941395, 0.8930312395095825, -0.16932858526706696, 0.5889033079147339, -2.1621131896972656, 0.11773654818534851, -0.999265193939209, -0.02837473526597023, 1.2794667482376099, -0.3515792489051819, 0.522208034992218, 0.6449486017227173, 0.8681407570838928, 0.2528716027736664, 1.8383567333221436, 0.010689656250178814, -0.5534741878509521, 0.4617471694946289, -0.5067040324211121, 1.6526442766189575, -0.7705622315406799, -0.5968149900436401, 0.23670274019241333, 1.0571027994155884, 1.0957690477371216, -0.7773667573928833, 1.860844373703003, -0.03776685893535614, -0.5330691337585449, -1.7346001863479614, -0.9644150733947754, -0.850253164768219, 0.4097408950328827, -1.6264287233352661, 1.0017386674880981, -2.58951735496521, 0.6351816058158875, 0.020115207880735397, 0.08628668636083603, -0.17225423455238342, -1.04156494140625, -1.629131555557251, -0.7910057902336121, 0.24790143966674805, -0.5542173385620117, 1.1951502561569214, -1.2490766048431396, 0.005944846197962761, -1.5909847021102905, -1.4833019971847534, -2.3597724437713623, 0.010565401054918766, 0.9427584409713745, -0.8514484763145447, 1.7857486009597778, 0.4316338896751404, -0.8208759427070618, -0.553451657295227, 0.1255318820476532, 0.29009994864463806, -0.14045293629169464, 0.38556960225105286, 1.7012678384780884, -0.2971183657646179, -1.2090983390808105, -0.10038655996322632, -0.8327974677085876, -1.5389658212661743, -0.10046549141407013, -0.4096376597881317, 0.18366952240467072, 0.6144658327102661, -0.5906768441200256, 1.10517418384552, -1.0896787643432617, -0.7578331828117371, -1.8439793586730957, 1.2020432949066162, 0.77421635389328, -0.2682482898235321, -0.13256016373634338, -0.06278595328330994, 0.6497988700866699, -3.2264528274536133, 1.354777216911316, 0.8329572081565857, -0.041222214698791504, -1.465641975402832, -0.7882899641990662, -0.46053221821784973, 0.2909559905529022, -0.3458474278450012, 1.0092612504959106, -1.213873028755188, -0.37738439440727234, -0.8676493167877197, -0.554970383644104, 1.3163625001907349, 0.7842403650283813, 0.25186988711357117, -0.3316700756549835, -1.28728187084198, 0.49661579728126526, -1.7592672109603882, -1.6063259840011597, 0.6847410202026367, -0.5839448571205139, -0.14227885007858276, -0.8113439083099365, -1.4646013975143433, -0.605222225189209, -0.214421346783638, 0.980857253074646], [-0.08508951961994171, 0.8468018770217896, -0.9215655326843262, 0.4324517250061035, 0.10958800464868546, 0.40081116557121277, -0.6698177456855774, 0.6772059798240662, 0.2692406177520752, 0.7094008922576904, -0.8653149008750916, -0.2205725610256195, 1.2123503684997559, 0.5462415814399719, 2.041062355041504, -0.8509116172790527, -0.7814292311668396, -0.7811484336853027, -0.7087576389312744, -0.29341834783554077, 1.2090344429016113, 0.5622250437736511, -0.4028877317905426, -0.5042749643325806, -0.3570401072502136, -0.6465367674827576, -1.4203382730484009, 0.1204342395067215, 0.7504048347473145, 0.5650613903999329, -0.8962080478668213, 0.01865188032388687, 1.6311016082763672, 0.6299808621406555, 0.13897520303726196, -0.2606738805770874, 1.4086054563522339, 0.858785092830658, -1.6404590606689453, -0.05353628471493721, 0.10384751856327057, -1.0997817516326904, -0.30706796050071716, 1.3444558382034302, 0.21578766405582428, -0.691877543926239, 0.25663575530052185, 0.6315252780914307, 0.059303756803274155, 1.6866846084594727, -0.5763401389122009, 0.18083296716213226, 0.7235840559005737, 1.6254909038543701, -0.10971151292324066, 0.38858792185783386, 0.482840359210968, -1.1834038496017456, 0.1144762933254242, -0.1755613386631012, 0.4615514576435089, -0.14761246740818024, 0.16821293532848358, 0.8172217607498169, -0.534338116645813, -0.437857985496521, 0.9764763712882996, 0.28838518261909485, -0.3015468120574951, 1.804304838180542, 0.9276915192604065, 0.31604084372520447, -0.7191008925437927, -0.17324994504451752, 0.49660536646842957, 0.9338545203208923, 0.38444867730140686, -1.4543023109436035, -0.8098980784416199, -0.8727250695228577, -1.541874885559082, -0.5270071029663086, 0.8050128817558289, -0.12289673089981079, -0.7450040578842163, 1.2699931859970093, 0.3415839970111847, -1.7051149606704712, -0.8107427954673767, 0.5642585754394531, 0.6432709097862244, -0.7485646605491638, 1.4974050521850586, 0.35051074624061584, 0.5080569982528687, -0.006429317872971296, 1.917206048965454, 0.4946308434009552, -0.6162309050559998, -0.5799590349197388, 1.2615399360656738, -0.23702210187911987, -0.06312009692192078, -0.5580889582633972, -0.5563270449638367, -1.281987190246582, 1.4266204833984375, 2.040587902069092, 0.940904974937439, -0.1449957638978958, 0.7725144624710083, 1.6851189136505127, -0.30858156085014343, -0.4666576683521271, -0.12450617551803589, 0.11619818955659866, -0.4165828227996826, -0.35312506556510925, 0.2649346888065338, -1.0044928789138794, -0.328843355178833, -0.4313098192214966, -1.2848094701766968, 1.0049171447753906, 0.6798079609870911, -1.9174407720565796, 1.2538087368011475, -0.05682902783155441], [1.412494421005249, 1.0912654399871826, -1.0501641035079956, -0.6353049278259277, 0.3777904510498047, -2.215938091278076, -0.8418809771537781, 0.4861868917942047, -0.09519325941801071, 1.3596447706222534, -0.5360100269317627, 0.10731713473796844, -0.5131022334098816, 0.9210504293441772, -0.5588303208351135, 0.9774694442749023, -1.0273680686950684, -0.8431276082992554, 0.2475782185792923, -0.023551935330033302, 0.17545075714588165, 0.6848441958427429, 0.1949443817138672, 0.049822766333818436, 2.048088788986206, 0.4214794337749481, 0.4155745208263397, 0.17289644479751587, 1.0117907524108887, -0.6704246401786804, 0.6857106685638428, -0.5311013460159302, -0.4288206100463867, -0.19897526502609253, -0.6451416611671448, 0.9051427245140076, 2.326864004135132, -1.6216577291488647, -2.1349704265594482, 0.8202924132347107, -1.056068778038025, 0.9268821477890015, -1.5651909112930298, -0.5830850601196289, 0.37708476185798645, -1.5810906887054443, -1.7597078084945679, 0.03520108014345169, 1.1828933954238892, 0.5816161036491394, -0.06031837314367294, -0.45286181569099426, 0.18688097596168518, -0.17346037924289703, 0.9420295357704163, 0.4934123158454895, 2.2557592391967773, -0.6194261312484741, 0.8091097474098206, -1.0531013011932373, -1.1142089366912842, -0.46175938844680786, 0.5414065718650818, -0.4438993036746979, -2.272432804107666, -1.283521294593811, -0.8184216022491455, 1.112058162689209, -1.5806604623794556, -0.11318162083625793, 0.34096094965934753, -0.6396588087081909, -0.8604435920715332, -0.5602648854255676, -0.31593167781829834, 0.7131472826004028, -0.2544354796409607, 0.18499648571014404, -0.06666356325149536, 0.0446406751871109, -1.1975020170211792, -1.0685248374938965, -0.9243118166923523, -0.3442256450653076, -0.11481279879808426, -1.7793772220611572, -0.9894614815711975, -1.0612682104110718, -0.9679082632064819, -0.03448386490345001, -0.056022245436906815, 0.5927648544311523, 0.4029534161090851, -1.0957884788513184, 1.2244064807891846, 0.2630225121974945, 0.17647968232631683, -1.1208375692367554, 0.7547159790992737, 1.5344825983047485, -0.11288601905107498, -0.28560081124305725, -0.9969291090965271, -0.8800095915794373, -0.6753993034362793, 0.7804689407348633, 0.4246883988380432, -0.9459370374679565, 1.1158616542816162, -1.1352301836013794, 0.6517556309700012, 1.1879056692123413, -0.6156936883926392, -0.8101810216903687, -0.22147811949253082, -1.2577284574508667, -1.199222207069397, 0.4092104136943817, 0.6459028720855713, -0.6577168107032776, -0.9565438032150269, 0.07948759943246841, -0.36368128657341003, -0.3701601028442383, 1.978932499885559, -1.1445801258087158, -0.712485134601593, 0.22102971374988556], [0.2900997996330261, 0.19131647050380707, -1.744175672531128, -1.6644470691680908, 0.9306942224502563, -0.459711492061615, -0.5969265103340149, -0.4648151695728302, 1.952834963798523, 0.5984652638435364, 0.21557928621768951, 0.5812435150146484, -0.7254699468612671, -1.4332358837127686, -0.11357622593641281, 1.2030097246170044, 0.29624947905540466, 0.8703567981719971, 0.5039858222007751, 1.5453892946243286, -1.9173327684402466, 1.282596468925476, -1.1557101011276245, -0.09098149836063385, 0.021841635927557945, 0.014815703965723515, 0.5778564810752869, -0.7286205291748047, 1.0243312120437622, -1.014689326286316, -1.5194921493530273, 0.09592137485742569, -0.4812304377555847, -0.46225282549858093, -0.07368078827857971, -0.04309575632214546, -0.019108125939965248, -0.9169886112213135, -2.5976903438568115, -1.3252077102661133, -0.7294141054153442, -1.7705738544464111, -0.6760648488998413, -0.05193435400724411, -0.4535287022590637, 1.4516593217849731, -0.29711446166038513, -0.31324222683906555, 0.11271946877241135, 0.09948384016752243, -0.32602769136428833, -1.2787134647369385, -1.9219670295715332, 0.3742511570453644, -0.989344596862793, -1.2849477529525757, -0.9602864384651184, -0.7419907450675964, -1.152853012084961, -0.907235324382782, -0.45019224286079407, 0.037717055529356, -0.8826646208763123, -0.9413135051727295, 1.1882232427597046, 1.01716947555542, -0.7637490034103394, -0.2931636571884155, -0.045276299118995667, -0.686393141746521, -0.2623271346092224, 1.950594425201416, -0.5458753705024719, -1.31844162940979, -1.588672161102295, 0.6367091536521912, 1.055423378944397, -1.4870259761810303, -0.6582580208778381, -0.28174087405204773, -0.48785465955734253, -0.8117080926895142, 0.6879262328147888, -0.13931842148303986, 1.026090383529663, -0.46917206048965454, -3.679067611694336, -0.0618712455034256, 1.4785819053649902, 0.17962029576301575, 0.4756004512310028, -2.2624588012695312, 0.7984906435012817, -1.2872226238250732, 0.37922143936157227, 0.2572287619113922, -0.47148528695106506, -0.003956148866564035, 0.42988690733909607, -0.4098801016807556, 0.34415188431739807, 0.26966592669487, 0.3000487685203552, 0.2767840623855591, 1.397519826889038, -0.8218969106674194, 0.6658868193626404, 0.09921339899301529, -0.09269390255212784, -0.7627812027931213, -0.70573890209198, -0.47620251774787903, 0.9097482562065125, 0.3260285258293152, 0.6198534965515137, -1.5532439947128296, -1.1376311779022217, 0.7518377304077148, 0.46312880516052246, 0.8186694979667664, -0.6045809984207153, 1.5531195402145386, 0.34230178594589233, -0.6443436145782471, -0.08300129324197769, -1.3761159181594849, 1.376729965209961, 1.1982347965240479], [0.4999066889286041, 0.5340954661369324, 1.380843162536621, -0.5515047907829285, -0.1524970382452011, -0.7688418626785278, -0.7111241817474365, -0.16001243889331818, -0.05955466628074646, 0.48364636301994324, 0.06565890461206436, -1.4587905406951904, 0.13147471845149994, -0.8099920749664307, -1.1152647733688354, 0.6049820184707642, 0.13386327028274536, 0.552574098110199, 0.2070057988166809, -0.6908044219017029, -0.6291459798812866, -0.46171990036964417, -1.5826090574264526, -0.907434344291687, 0.7983424663543701, -0.2078579217195511, 0.14821328222751617, 0.7030150294303894, -1.6953599452972412, -0.7823086380958557, -1.3194667100906372, 1.0838673114776611, 0.3454430103302002, 2.006239891052246, -1.1252050399780273, 0.31036239862442017, -0.420954167842865, -0.9160901308059692, 0.16483759880065918, 0.002691546455025673, -1.1617038249969482, -1.0248591899871826, -0.34003573656082153, -0.2788456380367279, -0.3642967641353607, -0.5800682902336121, 1.1773127317428589, 0.5954756140708923, 0.05630958080291748, 0.9356659054756165, -0.9700098633766174, -0.22011685371398926, -0.627246081829071, 0.010755983181297779, -0.7896903157234192, 1.4141212701797485, -0.4810485243797302, -0.6662404537200928, -1.7527724504470825, 0.25594329833984375, -1.6668463945388794, 1.104408860206604, 1.4654630422592163, -1.1177953481674194, 0.7062392234802246, 0.021130260080099106, -1.1385339498519897, -2.363480567932129, -1.055189609527588, 1.0897434949874878, -1.0709093809127808, -0.25432875752449036, 0.34756380319595337, 0.1635531634092331, 0.0828266367316246, -1.504583477973938, 0.5314857959747314, -1.2175348997116089, -0.2657012939453125, -0.3768883943557739, 0.5808904767036438, -0.19178485870361328, 0.12933847308158875, -0.997796356678009, -1.6418707370758057, -0.454620361328125, -0.9966447353363037, -0.07866879552602768, -0.36556246876716614, 1.0554710626602173, -0.0016404371708631516, 0.4074570834636688, 1.0242739915847778, -1.5967012643814087, 0.9111329913139343, -0.23578478395938873, -0.991489052772522, 0.25036919116973877, -0.10830529779195786, 0.8364410400390625, 0.9179093837738037, 0.21494625508785248, 0.18862153589725494, -1.1657792329788208, -1.2801893949508667, 1.5322437286376953, 1.1590325832366943, 1.1233221292495728, -0.3253464698791504, -0.28238487243652344, 0.05910815671086311, 0.6335570216178894, -0.9036139845848083, -1.2383148670196533, 0.3628619909286499, 1.430683970451355, -1.1550214290618896, -0.7418856620788574, 0.6494945883750916, -0.45002856850624084, 0.30333176255226135, 0.7568125128746033, 0.6641928553581238, -2.610621452331543, 0.3174094557762146, 0.4087132513523102, 0.8342999219894409, 0.1526113599538803], [-0.48925912380218506, 0.9609582424163818, 0.1312279850244522, -1.5423389673233032, 1.0247361660003662, -0.2566244900226593, -1.5310564041137695, 1.5256181955337524, -1.4615981578826904, -0.8564333319664001, -0.3707873821258545, -0.12323644757270813, 0.7056399583816528, -0.41040298342704773, 1.4303548336029053, 0.11987485736608505, -0.9550960063934326, -0.15348148345947266, 1.380454421043396, -0.2671393156051636, 0.08435545116662979, -1.5248796939849854, 1.1982673406600952, -0.33738330006599426, -1.3933113813400269, 0.6118906736373901, -0.05156252533197403, 0.9112042188644409, -0.6932428479194641, -0.5575324296951294, -0.2940944731235504, -0.006976124830543995, -0.8622162342071533, -0.11388633400201797, -1.0774387121200562, 1.7727352380752563, -1.1729236841201782, -0.8477965593338013, 0.27140969038009644, 0.9610301852226257, 0.6351974010467529, -0.5737982988357544, 1.41275155544281, 0.5872961282730103, 2.059082269668579, 1.4870648384094238, 0.44835612177848816, -2.8328800201416016, -0.24249625205993652, -0.9570533633232117, 0.621071994304657, -0.15235885977745056, 1.381799578666687, -0.09207219630479813, -0.9038788080215454, -0.9594529271125793, 3.061572551727295, 0.03662882000207901, -1.1750868558883667, 1.187714695930481, 0.19380803406238556, -0.008611306548118591, -2.5464065074920654, -0.34344181418418884, -1.438567876815796, 0.5365654230117798, -2.3207480907440186, -0.6451183557510376, -0.45769786834716797, -0.07501204311847687, 0.8664109110832214, -0.8536257743835449, 0.2890034019947052, 1.323438286781311, -0.6627035140991211, 2.285874128341675, -1.0431792736053467, 0.6008399128913879, -1.0820581912994385, -1.9972950220108032, -0.9144842624664307, -0.4797173738479614, -0.19524872303009033, 1.600008249282837, -0.6887297034263611, -1.0191280841827393, 1.7461967468261719, -1.0016013383865356, 1.4483493566513062, 0.7231431007385254, -0.0731273964047432, 0.04408338665962219, -0.4511262774467468, -1.5606368780136108, 1.2100151777267456, 0.5379788875579834, -1.7628980875015259, 0.44048577547073364, -0.9001640677452087, 0.37817883491516113, -0.5755505561828613, 0.870158314704895, -0.6603551506996155, -0.16127757728099823, -1.6941595077514648, -0.5758056044578552, 0.4140796959400177, -0.2654252350330353, -0.5217928886413574, -0.5803544521331787, 0.39547380805015564, 0.2956898510456085, 0.7914918661117554, -1.479466438293457, 0.5878609418869019, 0.5770025849342346, -0.8191671371459961, -0.11390262097120285, -1.2435287237167358, -1.7914316654205322, -1.0203020572662354, 0.7134718894958496, 0.942359983921051, 0.1745205968618393, 1.3361660242080688, 0.05652816966176033, -0.8263663053512573, 0.17807736992835999], [-1.1835073232650757, 0.7121251225471497, 0.44221383333206177, -1.6745882034301758, -2.599170684814453, 0.6139664649963379, 0.5547922849655151, 0.9561732411384583, 1.1346465349197388, -1.5547044277191162, 0.013513066805899143, -0.4133281111717224, 1.0621531009674072, -0.7798900604248047, -0.9213404655456543, 1.3508460521697998, 1.764334797859192, -0.3976220190525055, -1.156226634979248, 0.32617974281311035, -1.181880235671997, -1.3093982934951782, -0.5177274346351624, -2.4021739959716797, -0.877681314945221, 2.589202404022217, -0.6640554666519165, -0.0561661534011364, 0.9360652565956116, 0.5328702330589294, -0.4307299256324768, 0.539053201675415, -1.2014224529266357, -1.766678810119629, 1.0543479919433594, 2.5025596618652344, 0.7436617016792297, -0.1452857255935669, 0.8754297494888306, -1.0158673524856567, 0.7199941277503967, -0.06942673027515411, 1.3691654205322266, -0.7201223373413086, -0.46419668197631836, -0.5268737077713013, 1.3552474975585938, 0.9994326233863831, -0.45690903067588806, -1.2325634956359863, -0.08402916043996811, 1.7066988945007324, -0.8147334456443787, -1.5248104333877563, 0.05268010497093201, -1.1935089826583862, -1.3138707876205444, 0.5719462037086487, -0.48558634519577026, -0.6415722966194153, 1.1946978569030762, 0.3782348036766052, -1.5966516733169556, 0.7637073397636414, 0.8130876421928406, 0.3274577260017395, 0.19178877770900726, -0.36600056290626526, 2.0459585189819336, 0.45933204889297485, 0.16268280148506165, -0.060990095138549805, -2.1815547943115234, -1.4291530847549438, 0.78989177942276, 0.5232966542243958, -1.078235387802124, 0.14626047015190125, 0.6144775152206421, 0.27357637882232666, -1.0151309967041016, 1.8954743146896362, -1.961732268333435, 0.9738766551017761, -1.068927526473999, 0.09527113288640976, 0.028622804209589958, 0.057453423738479614, 1.3916996717453003, -0.7890096306800842, -0.4807688593864441, -0.4633495807647705, 0.6155312061309814, 0.880549430847168, 1.3953449726104736, 0.5234401822090149, 2.529512643814087, -0.47005802392959595, 1.2855725288391113, -0.9511357545852661, -0.45628511905670166, 0.9780234694480896, 2.2433669567108154, -0.33796370029449463, 0.5040194988250732, -1.648708462715149, -0.6757015585899353, 1.6213527917861938, 0.45123687386512756, 0.518153965473175, 0.23886016011238098, 0.05351715534925461, 0.7681190967559814, 0.08314895629882812, 0.681640088558197, -1.17740797996521, 3.0192015171051025, 0.07067758589982986, 0.16166500747203827, 0.7620725035667419, 0.7091426849365234, -1.6858831644058228, -0.07550132274627686, 0.9938201904296875, -0.8777374029159546, -1.4049829244613647, 0.714657723903656, 0.33172109723091125], [1.0324293375015259, -1.1614550352096558, -0.215937539935112, 1.27390456199646, -0.4858584403991699, 0.8030405044555664, 0.6755856275558472, -0.8472784757614136, 0.8162170052528381, -0.22944101691246033, -0.1762549877166748, 1.4238373041152954, -0.4937637746334076, 1.6330392360687256, -1.0780308246612549, -0.9933891296386719, 0.3554883897304535, -0.14587010443210602, 1.958661437034607, 0.0443294495344162, -1.1566007137298584, -1.847056269645691, 0.7209595441818237, -0.6346396803855896, 2.0950253009796143, 0.15056119859218597, -0.12640812993049622, -0.5943643450737, 0.41006147861480713, -0.9876232743263245, -0.41666316986083984, -0.12047957628965378, -1.44920814037323, 1.697641134262085, -0.7583232522010803, 0.9059156775474548, -0.2561038136482239, 0.33088386058807373, 1.1391609907150269, -0.25124454498291016, -1.0057564973831177, 1.2756599187850952, -1.7679266929626465, -0.7447313070297241, 0.9057657122612, -1.7236407995224, 0.5509727597236633, 0.2696024477481842, -0.20378638803958893, 0.16777531802654266, 1.6586662530899048, -0.9579013586044312, -0.0305153988301754, -0.677994430065155, -0.1894683986902237, 0.23738005757331848, 0.3112664818763733, 0.34099528193473816, -0.9705223441123962, 0.5726091861724854, 1.365388035774231, -0.4709508717060089, 1.5959317684173584, 0.9256798624992371, -0.04803925380110741, -0.9729027152061462, -0.49905091524124146, -0.7377269268035889, 0.42962753772735596, -1.0250338315963745, -1.2637908458709717, 0.550411581993103, 0.6873824000358582, -1.269034743309021, 1.6445080041885376, 2.5951383113861084, 1.5248128175735474, -0.07010867446660995, -0.42355772852897644, 0.5087162852287292, -1.1244704723358154, -1.6986815929412842, -0.4247811436653137, 1.3401395082473755, -0.6380009055137634, 1.0081440210342407, 0.9662337303161621, 0.13034625351428986, -0.5124287605285645, 2.6010255813598633, 1.1702030897140503, 0.036915332078933716, 2.2415523529052734, 1.176393985748291, -0.5231649279594421, -0.8306981921195984, 0.09253710508346558, -0.7395046353340149, 0.6139577627182007, 0.9134973287582397, -1.4333076477050781, 0.29823264479637146, -0.08447592705488205, 0.42992791533470154, -0.6346613764762878, 1.4704116582870483, -1.148890733718872, 0.3647836446762085, 0.2172461301088333, 1.018855333328247, 1.9311256408691406, 0.4941296875476837, -0.7047876715660095, 1.17507004737854, 1.7236741781234741, -0.7484036087989807, 0.45306745171546936, -1.1826492547988892, 0.7043898701667786, -0.9514672160148621, 0.7416298389434814, 1.0309070348739624, 0.31860437989234924, 0.7968588471412659, -0.39031854271888733, -0.7885724902153015, -1.2804677486419678, -1.0430222749710083], [-0.9120847582817078, 1.450722336769104, -2.3593411445617676, 1.1796975135803223, 0.08050097525119781, 1.567175269126892, -1.135080337524414, 1.0651671886444092, -0.8644511103630066, 0.41893449425697327, -0.29788827896118164, 1.1290581226348877, 0.9615883827209473, -0.4737183451652527, 0.43582162261009216, -0.03733493387699127, -0.30540093779563904, -1.7621917724609375, -0.3181056082248688, -0.02595771662890911, -2.4313933849334717, 0.8888700604438782, 1.1231104135513306, 1.0687508583068848, -0.23578347265720367, 1.7075140476226807, 1.006471037864685, -0.49691882729530334, 1.5147336721420288, 0.032140593975782394, -0.6725347638130188, -0.7337876558303833, 1.8828513622283936, 0.23144713044166565, 1.662001371383667, -1.1795364618301392, -1.3636767864227295, -1.2202173471450806, -2.4003870487213135, -0.9248567223548889, -0.7210448980331421, 0.5230192542076111, -1.1548174619674683, 0.028790831565856934, -0.8907812833786011, 1.5438780784606934, -1.1850144863128662, 0.9057270288467407, 2.3955883979797363, -1.3783282041549683, -1.3103216886520386, 0.6613067388534546, 0.2535344660282135, -0.320583313703537, -0.1878560185432434, 0.07986826449632645, -1.0048305988311768, 1.6646909713745117, 0.2032678872346878, -0.5395162105560303, 0.12219356000423431, 1.3252345323562622, -0.5224721431732178, 1.3979521989822388, -0.7747567892074585, -0.6817591190338135, -1.572481632232666, -0.13461020588874817, -1.1515321731567383, 0.9171082377433777, -0.7491622567176819, 2.8520874977111816, -0.7932415008544922, 1.741987705230713, 0.368911474943161, -0.8467163443565369, 0.8284722566604614, 1.9939478635787964, -1.129986047744751, 0.5416487455368042, 0.11778150498867035, -0.30942878127098083, 0.3851335644721985, -1.3111850023269653, 0.6479909420013428, 1.4145550727844238, 0.5982085466384888, 0.34297534823417664, 0.9900728464126587, 0.45098158717155457, -0.08139988780021667, 0.6518352627754211, 0.9609126448631287, 0.07154150307178497, -0.11605044454336166, 0.15353035926818848, -0.3654860556125641, -0.8988369703292847, -1.1756930351257324, 1.529963731765747, -0.822324812412262, -0.543924868106842, -0.03918290510773659, 0.01137456949800253, -0.2638038396835327, -0.0540769137442112, 0.7434355020523071, -0.7375478148460388, -1.3920496702194214, 0.27513036131858826, 1.7271381616592407, -0.03703460097312927, 0.4638817012310028, 1.497531771659851, -0.20880018174648285, -1.016237497329712, 0.966060221195221, 1.384908676147461, 0.864450991153717, 0.7278337478637695, -0.5447567701339722, -1.1040151119232178, -0.3962826430797577, -2.7029850482940674, -0.6120663285255432, 0.8105989694595337, 1.2188210487365723, 1.134225606918335], [-0.8689504265785217, -0.2746170461177826, -0.8223097324371338, -0.39464297890663147, 1.0478838682174683, -0.36201411485671997, -0.037818294018507004, 0.4963311553001404, -2.574706554412842, -0.3516154885292053, -1.8593114614486694, -0.19420720636844635, 0.48298609256744385, 0.5311814546585083, 0.2640577256679535, 0.09347958862781525, -0.1740589290857315, -1.7807908058166504, -0.5296299457550049, -1.4487229585647583, 1.030094027519226, -0.15302374958992004, -0.536535918712616, 0.971869170665741, 1.540683388710022, -0.45310673117637634, 0.1565098911523819, -2.5114786624908447, 1.4912612438201904, 0.12223386764526367, -0.24481013417243958, -0.8415794372558594, 0.2529292404651642, 0.16218450665473938, -0.34066417813301086, 0.5673283934593201, 1.3385289907455444, -1.0877947807312012, -0.6869934797286987, -1.067476749420166, 0.566511332988739, 1.1717594861984253, -1.02992844581604, 0.7665131092071533, -0.562671422958374, 0.9118315577507019, -1.3567880392074585, 0.902833878993988, -0.14582635462284088, -1.2717304229736328, 0.04840066283941269, -0.46458908915519714, -1.7222142219543457, -0.67974853515625, -0.1262965053319931, 2.502593994140625, 0.40631213784217834, -0.4554709494113922, -0.2054283171892166, 0.999718427658081, -1.6290514469146729, 1.0141420364379883, -0.31973543763160706, 0.0420956127345562, -1.5856279134750366, -0.2804751992225647, -1.1253443956375122, 0.4396101236343384, -0.8757112622261047, -0.04230429232120514, 0.2875065207481384, -0.9036579728126526, 2.4758105278015137, -0.8208642601966858, 0.6377661228179932, -0.7842898368835449, 0.005853333510458469, 2.5525734424591064, 0.6572259068489075, 0.9653492569923401, 0.061435043811798096, -0.24059396982192993, -1.543210744857788, 0.711487889289856, 0.7706249952316284, -1.0533405542373657, -0.8451381921768188, 0.5910260677337646, 2.1634225845336914, -0.9612704515457153, 1.0100560188293457, 1.0666038990020752, 0.6340692639350891, -0.11551915109157562, 0.47529125213623047, 0.3851662278175354, 0.10310228168964386, -0.44654905796051025, 0.7961313128471375, 0.7988362312316895, 1.262214183807373, 0.13944892585277557, -0.4562996029853821, -1.052384614944458, -1.0857166051864624, 0.5890406370162964, 0.8332500457763672, 0.7478470206260681, -1.1786301136016846, 1.544374942779541, -1.4662889242172241, 0.3587053716182709, -0.1006108894944191, -1.7641311883926392, -0.7037970423698425, 1.3475204706192017, -0.7358949780464172, 0.3660666346549988, -1.4863072633743286, -0.6329774856567383, 0.5960929989814758, 0.6229690909385681, 0.6846696138381958, -2.3608312606811523, 0.14679737389087677, -0.03405499830842018, 1.1740927696228027, 0.3114299774169922], [-0.9477525353431702, 0.16052068769931793, 0.16339729726314545, -0.8351878523826599, -2.7690231800079346, -0.5754214525222778, 0.3571224510669708, 1.690970778465271, 0.5610281825065613, 0.10190989077091217, 0.250161349773407, 0.5195735096931458, -1.483298420906067, -0.9443438649177551, -0.8785057663917542, -0.8728160858154297, -0.8048092126846313, -1.0675143003463745, -2.4250190258026123, -0.16647489368915558, 0.0971095934510231, -1.1949422359466553, 0.45455819368362427, -0.053855717182159424, -0.1285388171672821, 0.9659866690635681, 0.028577201068401337, -0.9012709856033325, -0.09970220923423767, 0.38537490367889404, -0.14368808269500732, 0.152807354927063, -0.18708068132400513, -1.3645687103271484, -0.7250715494155884, -0.3247777223587036, 1.3870331048965454, 0.9973262548446655, 0.030234968289732933, 0.9093000292778015, 0.7084673643112183, -0.28600284457206726, 0.3395053744316101, -0.9227783679962158, -1.451178789138794, -0.22092880308628082, -1.3877514600753784, 1.3709702491760254, -1.2208521366119385, -1.2075419425964355, -0.851950466632843, -0.9843329787254333, 1.5502800941467285, 0.10250777006149292, 0.8901785612106323, 0.24919749796390533, 1.1038670539855957, 0.507330596446991, -0.9868624806404114, -0.1642046719789505, 0.05485948547720909, -0.7366952300071716, -2.254603385925293, 0.7892338633537292, 1.4444390535354614, -0.41406458616256714, 3.358372926712036, -0.010319527238607407, 0.07402769476175308, 0.473279744386673, -0.9137392640113831, -0.8939041495323181, -1.2978007793426514, -0.024154623970389366, 0.24950739741325378, -1.3512030839920044, -1.1046860218048096, -0.13942454755306244, 0.8957763910293579, -0.4018552899360657, 0.048725757747888565, -0.2427828162908554, 1.0347472429275513, 1.204028606414795, 0.4489041864871979, 0.08888986706733704, -0.3542596399784088, -0.8474798798561096, -0.10219905525445938, -1.1551241874694824, -0.3414446711540222, 0.08130205422639847, -1.3192466497421265, -1.132482886314392, -1.2081950902938843, -0.034047603607177734, -0.4322066009044647, 1.7098275423049927, -0.6920502781867981, -0.739663302898407, -0.4184371829032898, 0.4642229378223419, 1.9240093231201172, 0.457051545381546, 1.2303466796875, -1.4416910409927368, -0.3152574300765991, -0.9880529642105103, -0.3340420722961426, 1.5266776084899902, -0.6179490089416504, 0.0023770141415297985, 0.9314723610877991, 1.048365831375122, 0.17482890188694, 1.5295766592025757, 0.9037837386131287, 1.0840035676956177, 0.12487722933292389, -0.1454494148492813, -1.2848621606826782, -1.1690187454223633, -0.1785375475883484, -0.37575629353523254, -0.24955151975154877, -0.2917926609516144, -1.8658263683319092, -0.8685834407806396], [-1.0836443901062012, -0.7573887705802917, 2.23093581199646, -0.08368716388940811, 0.01066607516258955, -0.467582643032074, -1.875239372253418, 0.1757371872663498, 0.2042006403207779, 0.9192975163459778, 0.6742594242095947, -1.880578637123108, -0.6833991408348083, -1.0020804405212402, 1.626721978187561, -0.633516788482666, 0.029771216213703156, 0.03724227100610733, 0.6809834241867065, -0.4442664086818695, -0.7554796934127808, 1.007670283317566, -1.6740953922271729, -0.673401415348053, -0.06042635440826416, 1.28988778591156, 0.9110578894615173, 0.4260658621788025, 0.2355007380247116, -1.0915240049362183, 0.9120579361915588, 0.16800497472286224, -0.3341251015663147, -0.03355412557721138, 0.3294014632701874, 0.3022836148738861, -0.36600542068481445, -0.08779776096343994, 3.6343178749084473, 0.5816145539283752, -1.2798938751220703, 0.8433367609977722, 2.2120981216430664, -0.12018678337335587, 0.7269144654273987, 0.49549293518066406, 0.9328096508979797, -1.0032763481140137, 0.748897910118103, 0.8577982783317566, 0.9386937618255615, -0.07262461632490158, -0.2273472100496292, -0.7315860986709595, 0.4257935881614685, 1.6053705215454102, -1.2227258682250977, -0.790221631526947, -0.3500225841999054, -0.1836978793144226, 0.22738374769687653, -2.415100574493408, -0.911518394947052, 0.16577234864234924, -0.699110209941864, -0.6919373869895935, -1.1250046491622925, -0.7072740793228149, 0.6686814427375793, -0.7780935764312744, 1.1918137073516846, -0.47134873270988464, -0.42114341259002686, -0.13949868083000183, 1.2272696495056152, 0.42698317766189575, 1.5914993286132812, -2.01488995552063, 0.8539997935295105, 1.5260772705078125, -0.8463367223739624, -1.5742589235305786, 0.5737765431404114, -1.3313634395599365, -0.7683023810386658, -1.39936363697052, -2.1210384368896484, 0.505627453327179, 0.6707954406738281, 0.32122084498405457, 1.3739573955535889, -0.25790950655937195, -0.6618574857711792, 2.0906991958618164, 0.5171453356742859, -0.854315996170044, -0.8970180153846741, 0.5361940860748291, -0.6539052128791809, -1.097030758857727, -0.9813016057014465, 0.9515073299407959, 0.15422731637954712, -2.2137551307678223, -0.9982426762580872, 0.17877110838890076, -0.6249642372131348, 1.1882407665252686, -0.5067428350448608, 0.709804117679596, 0.19396963715553284, 0.48940691351890564, 1.302432894706726, 1.714707612991333, -0.2296311855316162, 1.8498666286468506, -0.03295181691646576, 1.2536699771881104, 0.29280275106430054, 0.46158167719841003, 0.7784146666526794, 2.0832645893096924, -0.7639017701148987, 0.7633717656135559, -0.6316527724266052, 0.45881223678588867, 1.5510870218276978, -0.24369242787361145], [-0.31459328532218933, -2.0367636680603027, -0.33599650859832764, -1.0412061214447021, -1.8263838291168213, -0.5287734866142273, 1.0488630533218384, 1.8230922222137451, 0.9587706327438354, 0.9069522023200989, 0.42755332589149475, -0.38193899393081665, -0.921232283115387, -3.1388027667999268, -0.2965559661388397, 0.9019576907157898, -0.9181884527206421, 1.2776440382003784, -0.7445796728134155, 1.1332626342773438, -0.3382485806941986, -0.9491162300109863, -0.7258099317550659, -0.3921669125556946, 2.314566135406494, 0.016890501603484154, 0.0382838137447834, 0.7579405903816223, -0.7770668864250183, 0.5001147985458374, 0.12072101980447769, 0.47516632080078125, -1.7499992847442627, 0.8990859389305115, -0.05872562900185585, 0.6612188816070557, 1.4463372230529785, 0.6005052924156189, -0.28823593258857727, -0.7547862529754639, 1.2505508661270142, 1.120345950126648, -0.0055048177018761635, 0.20546361804008484, 0.3705083429813385, 1.3261092901229858, -0.15738876163959503, 1.5165473222732544, 0.13898006081581116, -0.7393558621406555, 1.3281203508377075, -0.07083253562450409, 0.1884646713733673, 0.018805982545018196, -0.1530657708644867, -0.9787166118621826, 0.042124830186367035, -0.9088263511657715, 0.9684000611305237, 0.6757180690765381, -0.2008449137210846, -0.5388196110725403, 1.3199435472488403, -0.8520290851593018, -1.7856695652008057, 1.1413593292236328, -1.9827322959899902, -0.8055394291877747, 1.994470238685608, -1.6903362274169922, 0.42734697461128235, -0.96957927942276, -2.2834620475769043, -0.4667263329029083, 0.4864130914211273, 0.10145718604326248, 0.48646923899650574, -1.606905460357666, 1.5674813985824585, -1.7199232578277588, 0.7359611392021179, -0.4490409791469574, -1.6480942964553833, 0.41202211380004883, 0.027708081528544426, 0.8045875430107117, 0.15235306322574615, -0.21018511056900024, 0.14955662190914154, 1.1686893701553345, -0.7318804860115051, -0.9859628677368164, 0.2942966818809509, 1.004874348640442, -0.7302395701408386, -1.6380640268325806, 1.7381436824798584, 0.9016152620315552, 0.2947673797607422, -2.4246456623077393, 1.3045048713684082, 0.7689491510391235, 1.51067054271698, -1.5131235122680664, -0.7501790523529053, -1.5320370197296143, -0.7466010451316833, -1.2075494527816772, -0.2013295292854309, -1.0048872232437134, 1.9221229553222656, 0.1130409762263298, -0.08655967563390732, 0.6431892514228821, 1.0575120449066162, 0.8276830315589905, 1.8113311529159546, 1.1142468452453613, 1.2121680974960327, -0.6498532891273499, -0.5488755106925964, -0.9844439029693604, 0.13836030662059784, 1.4194649457931519, 0.894100546836853, -0.5632196664810181, 0.3521643579006195, -0.8433828353881836], [-0.487956166267395, -0.9528456926345825, 0.5463895797729492, -1.5365509986877441, -1.7607978582382202, -0.0509113073348999, -0.9489482641220093, -0.2344321459531784, -0.04799247905611992, 2.076046943664551, -1.2362364530563354, 0.04217638447880745, 1.2453798055648804, 0.1964419037103653, -0.07354826480150223, 0.8968124389648438, 0.04607381671667099, -0.5488448143005371, 0.7203897833824158, 1.3329010009765625, -0.16599437594413757, -1.5731967687606812, -0.5817790031433105, 1.9174997806549072, 0.9463617205619812, -0.5991880893707275, -0.019233204424381256, -1.754293441772461, -0.1827675849199295, 0.7515720725059509, 1.0166683197021484, -0.2396925985813141, -0.9620257616043091, 0.4056514799594879, -1.7035481929779053, -1.0705379247665405, -1.1687744855880737, -0.868891716003418, -0.6826076507568359, 0.244212806224823, 1.0855917930603027, -0.9193931221961975, -0.17109344899654388, 2.062236785888672, -0.6052876710891724, -1.8359394073486328, -0.22931957244873047, -0.43718764185905457, -0.814658522605896, 0.8961652517318726, 0.7522242069244385, -0.45242899656295776, 0.7118297815322876, 1.19295334815979, 1.019230842590332, 0.5137546062469482, 0.6410251259803772, -1.2376359701156616, 0.7275769114494324, -0.7363953590393066, -2.2519400119781494, 0.837110161781311, 0.7803142666816711, 1.1105092763900757, -0.20413871109485626, -1.8026782274246216, 1.1665173768997192, 2.0269994735717773, -1.7309352159500122, -1.0130949020385742, -0.9446208477020264, 0.4906140863895416, 0.01985669881105423, 0.3949427008628845, 0.6291666626930237, -2.324821949005127, -0.7913597822189331, 0.6603208184242249, 0.20526206493377686, -0.32032689452171326, 0.541037380695343, 0.4677240252494812, 0.3123227655887604, -0.5884786248207092, 0.865424394607544, 0.37311050295829773, -0.10500217974185944, -0.29135268926620483, 0.875413715839386, -0.36227041482925415, 0.5452023148536682, 2.5225327014923096, -0.5688267350196838, -0.38357946276664734, 0.11751504987478256, 0.21114735305309296, 0.45322489738464355, 0.5376341342926025, 0.5848048329353333, 0.37889593839645386, -0.19360636174678802, 0.42375218868255615, -1.6232123374938965, -0.3261396288871765, -0.22748929262161255, 0.06888879835605621, 0.49886438250541687, 1.1753357648849487, 0.1755550652742386, -1.6923233270645142, -0.8772344589233398, 0.21194201707839966, 1.4606767892837524, 0.0028054192662239075, -0.2953278124332428, 0.47593221068382263, -0.669317364692688, -0.4987484812736511, -2.033099412918091, -1.1831979751586914, 0.5940155386924744, 0.20081353187561035, 1.8990908861160278, -0.5107250213623047, 1.4406236410140991, 0.3277500569820404, 0.7745837569236755, -0.6284968852996826], [0.21479374170303345, -0.40620118379592896, -0.8212169408798218, 1.2261375188827515, 0.7110751867294312, -0.693163275718689, -0.30803942680358887, -0.9085435271263123, -1.101157546043396, 1.1418406963348389, -0.4048633873462677, -0.25526729226112366, 0.23081794381141663, -1.1026124954223633, 0.6772746443748474, 0.7352557182312012, 1.1579657793045044, -0.3449596166610718, 1.0752590894699097, 0.6989462971687317, 1.8257746696472168, -0.11658232659101486, 1.2832131385803223, 2.130291223526001, 1.2637310028076172, 1.2040514945983887, 0.5011571645736694, 0.009234068915247917, -0.44568103551864624, -0.08495225012302399, -0.43304747343063354, -2.0702176094055176, 0.20514264702796936, 0.9038992524147034, 0.10800683498382568, 0.018398789688944817, 0.07434280961751938, 0.8581507205963135, 0.4864928424358368, 0.7542284727096558, 0.4282507002353668, 0.05358007177710533, 0.5984293818473816, -1.2483186721801758, 0.17586266994476318, -0.023505065590143204, 1.9069225788116455, 0.9832232594490051, -0.15781140327453613, -0.26605620980262756, 0.5120494961738586, 0.9355447888374329, 0.7960014939308167, -0.6368319392204285, 0.29247963428497314, 0.3408529758453369, 0.20101678371429443, 0.9763950705528259, -1.4003100395202637, 0.9260186553001404, 1.700813889503479, 0.3218652307987213, 1.2303745746612549, -1.8213250637054443, 0.37056422233581543, -0.5808091163635254, 1.576799750328064, -0.11329673230648041, 0.42052021622657776, -0.981731116771698, 1.112817645072937, 1.5286576747894287, 0.8547006845474243, 0.49022188782691956, 0.5928328633308411, -0.13327611982822418, 0.2815314829349518, -0.11675567924976349, 1.0900397300720215, 0.4643556773662567, -2.121703863143921, 0.917616069316864, 0.2712297737598419, 0.002573093632236123, -0.2570227086544037, 0.19145767390727997, -0.8282448649406433, 0.5138822197914124, -0.8987242579460144, -0.6229172945022583, 0.3573128581047058, 0.7815741300582886, 0.13370181620121002, -0.7935146689414978, -1.6991759538650513, -0.43374553322792053, 0.8021165132522583, -2.0589258670806885, 0.7431784272193909, -0.8303777575492859, 0.6859167218208313, -0.5800443291664124, -0.4309479296207428, -1.5575275421142578, 1.0105712413787842, -0.4445532262325287, 0.9737014174461365, -0.692768931388855, -0.20826612412929535, 0.5386514663696289, 2.587334632873535, 0.48074617981910706, 0.06400634348392487, -0.07080697268247604, 1.0337568521499634, -1.269263505935669, -2.1497814655303955, -0.9929943680763245, 1.3209911584854126, -0.6861065030097961, 1.4027100801467896, -0.6272496581077576, 1.5527681112289429, 0.8170902132987976, 0.2622532546520233, 1.1645565032958984, 1.0714678764343262, 0.8683826923370361], [0.9556649923324585, -0.04391658306121826, -0.4173509180545807, -0.6522276997566223, 0.16337214410305023, -1.1424616575241089, -0.4230096936225891, 0.9123860597610474, -0.8931049704551697, -0.6006821990013123, -0.2615429759025574, 1.55239737033844, -1.8039900064468384, -0.8377184867858887, -0.7696930170059204, 1.0249749422073364, 0.18801844120025635, -0.766228437423706, 2.7666256427764893, -0.4324679374694824, -0.49954351782798767, 0.8734753727912903, -1.007461667060852, 1.636337399482727, -0.22490419447422028, -0.6505895256996155, -0.4436732232570648, 0.7569763660430908, 1.7186940908432007, 0.4019233286380768, -1.3350862264633179, -1.6165391206741333, -1.4712390899658203, 2.1788365840911865, 0.5485052466392517, 0.14304617047309875, 1.0709774494171143, 1.7269366979599, -0.6124890446662903, 0.16590635478496552, -0.4740277826786041, -1.777622938156128, -0.795315682888031, 0.8533539772033691, 0.14624512195587158, -0.7325738668441772, -1.7642216682434082, -0.13895103335380554, 0.4834374785423279, -0.3396940529346466, -0.3092797100543976, 1.7839871644973755, -0.3712671101093292, 0.13946907222270966, -1.381393313407898, 2.145480155944824, 1.3611036539077759, -0.4637521207332611, 0.6967726945877075, 0.533798098564148, 0.01843511126935482, -0.5410475134849548, 2.1854166984558105, 0.3017579913139343, 0.0888894721865654, -0.4331018030643463, -0.07633557170629501, 1.5289628505706787, -1.698988437652588, -1.8645449876785278, -0.2037966549396515, -0.10939731448888779, -2.163771152496338, 1.3315118551254272, 0.1621176302433014, -1.042388916015625, -0.2048155516386032, 0.5357952117919922, 0.26127466559410095, -0.09787224233150482, 1.3634103536605835, -0.6355829238891602, -2.2847414016723633, 0.13738913834095, -0.23986051976680756, -0.6202138066291809, 0.13766515254974365, -0.9709419012069702, 0.3833056092262268, -0.7571335434913635, 0.9257636070251465, -1.0710580348968506, -0.01882063038647175, -0.8855639696121216, -0.7162607312202454, 0.17573225498199463, 2.1529390811920166, 0.28939810395240784, 1.3386527299880981, 0.12648046016693115, -0.7228606939315796, -0.9186837077140808, 0.4401589035987854, -1.9226778745651245, -1.0507493019104004, -2.6693332195281982, 2.117816925048828, -0.019326912239193916, -1.1452258825302124, -1.1045960187911987, -0.7843865156173706, 2.586501121520996, 1.0950847864151, -0.772487461566925, 0.6631252765655518, -0.1934179663658142, 0.2109318971633911, 0.09029021859169006, -0.12038371711969376, -1.3920233249664307, -0.8649480938911438, 0.4794329106807709, 0.8041195869445801, -1.1159526109695435, 0.7553095817565918, -1.4183032512664795, -0.10259480774402618, -2.086803436279297], [0.27167826890945435, 0.30296987295150757, 0.1900976598262787, 1.529467225074768, 0.6328993439674377, 0.34533897042274475, 0.08474551141262054, -1.1150895357131958, 1.1972118616104126, 1.077111840248108, 1.4526822566986084, -0.5081331729888916, 0.0051561687141656876, -0.46671053767204285, -0.8518803119659424, 0.9987821578979492, 1.346680998802185, 1.311681866645813, 0.14296138286590576, -0.6937257647514343, -1.1351863145828247, -0.24029308557510376, 0.13075187802314758, 0.1266704946756363, -0.5504571199417114, -0.583311140537262, -0.42911791801452637, 0.6193162798881531, 1.2869226932525635, -2.145571708679199, -0.44144701957702637, -0.2885880172252655, 0.438048779964447, -0.7867372035980225, 0.7269404530525208, 1.1216531991958618, 0.029149645939469337, 0.7334269881248474, 0.49614831805229187, -1.3549857139587402, -0.6382109522819519, -0.9048638939857483, -1.3193787336349487, 1.245666265487671, -0.8461575508117676, 0.6570461988449097, 0.340728223323822, -0.624776303768158, 1.3150180578231812, 0.16252867877483368, -0.6706214547157288, -1.3822789192199707, -0.734131932258606, 0.5699754357337952, 1.7543272972106934, 0.8877530694007874, -1.0524064302444458, -0.15189634263515472, 1.4459470510482788, -1.313200831413269, 0.7410101294517517, -1.079062819480896, 0.643395721912384, 0.5608199238777161, -1.0268042087554932, 0.14532101154327393, -0.7991641163825989, -1.587005615234375, 0.4669930040836334, 0.11657451838254929, 0.7787417769432068, -0.13905496895313263, -1.4311972856521606, 0.6893892288208008, 0.7810511589050293, 0.5996443033218384, -2.3721463680267334, 0.32231950759887695, -1.2004843950271606, 0.2495739609003067, 0.8514232039451599, 0.03464771807193756, -1.2686772346496582, -0.5038789510726929, -0.5175712704658508, -0.9969891309738159, 0.2108699381351471, 1.5933693647384644, 0.38888129591941833, -0.41067737340927124, -0.3434445858001709, 0.20011112093925476, 0.2127160131931305, 0.133962944149971, -0.6907526850700378, -0.12351852655410767, 0.5494455695152283, -0.5236321091651917, -0.4273117482662201, -0.8731732368469238, -1.095407247543335, -1.8187339305877686, 0.6770068407058716, -0.0003696621861308813, -1.8145464658737183, -0.7131937742233276, 0.19880925118923187, 0.766201376914978, -0.7718883752822876, -0.25268399715423584, -1.616958737373352, 0.5612823963165283, -0.6845535039901733, 2.2239723205566406, 1.6229476928710938, -0.302465558052063, 1.0342941284179688, 1.6607550382614136, -1.564324140548706, -0.94158935546875, 0.8946191072463989, -0.04874951019883156, -2.3956680297851562, -0.029706522822380066, -0.8309895396232605, 0.22530251741409302, -0.04334733635187149, 2.3668603897094727], [1.2914568185806274, -0.35496821999549866, -1.0024434328079224, -0.7916757464408875, 0.9329003095626831, -0.47998350858688354, 0.2821064591407776, -1.0067825317382812, 0.14661556482315063, -0.29833415150642395, 0.041945282369852066, -1.3963466882705688, 0.8882914185523987, 0.45859453082084656, 0.3150959312915802, 0.3331293761730194, 0.0032558580860495567, 1.7385376691818237, -0.09418335556983948, 0.6328367590904236, -1.411446452140808, 0.9723249673843384, -0.2108072191476822, 0.20680925250053406, -0.3223339021205902, -0.33656221628189087, -0.6846330761909485, -0.24006178975105286, -1.804364800453186, -0.20408880710601807, 1.9717732667922974, -0.2616361081600189, 0.34265878796577454, -0.09788691997528076, 0.13230152428150177, -0.8715699315071106, 0.34591144323349, 0.9347214698791504, 1.157841682434082, -0.822559118270874, -0.3067520260810852, 0.1554298996925354, -0.42871618270874023, -0.5621967315673828, -0.8154863119125366, -0.8855084180831909, -1.1259816884994507, -0.9551960825920105, -1.2114324569702148, -0.3166273236274719, -2.139953136444092, 0.29676780104637146, -0.6759907007217407, 0.5287061929702759, -1.6328140497207642, 0.4693102538585663, 0.8515399098396301, -0.6840108633041382, 0.6263846158981323, 0.12928703427314758, -0.8858048319816589, -0.1677556335926056, 0.2772899866104126, 1.964917540550232, -1.0189294815063477, 0.5090116262435913, -0.2459275871515274, -0.32612013816833496, -0.7469362616539001, -1.123869776725769, 0.09974849969148636, -1.5210367441177368, -0.6025691032409668, 0.029139846563339233, 2.280263662338257, 1.179084062576294, -0.1858517825603485, -0.3362036347389221, 0.08063830435276031, -2.3561999797821045, 0.7598434686660767, -0.03403971716761589, -0.8880711197853088, 0.9638166427612305, -0.11694132536649704, -0.9770529866218567, -1.4829905033111572, 0.6017250418663025, -1.8432410955429077, -0.8816326856613159, -0.13586720824241638, -0.006746856961399317, -0.37310901284217834, -0.3293705880641937, 1.8112919330596924, 0.527472198009491, 0.8624919652938843, -0.23495101928710938, 0.922637403011322, -0.24759972095489502, 0.5148532390594482, 0.14004741609096527, -0.8568695187568665, 0.9595872163772583, -2.0621981620788574, 0.4279019832611084, -0.3576062023639679, -0.9999728798866272, 1.1076295375823975, 1.9202104806900024, -0.9870116114616394, 1.5997707843780518, 0.670502781867981, -0.31186288595199585, 0.3604341149330139, -0.11859039217233658, 0.6341373920440674, -0.9806453585624695, 2.432573080062866, 0.30377060174942017, 1.7897965908050537, -0.8600634336471558, 1.7715699672698975, -0.4037119448184967, 0.5532687902450562, 0.45274800062179565, -0.15780793130397797, 0.07269364595413208], [-0.42144089937210083, 1.3409217596054077, -1.5696561336517334, 0.5682983994483948, -0.2079555243253708, 0.8923836350440979, -0.5317534804344177, -0.15489225089550018, 1.3771061897277832, 1.6214832067489624, 0.5514208674430847, -0.05371249467134476, -0.7280784249305725, -1.2305890321731567, 0.4108739495277405, 0.566504955291748, 0.5500465035438538, 2.204131841659546, 0.787712812423706, -0.010795067064464092, 2.5082032680511475, 0.9746017456054688, -1.2404295206069946, 0.9176639318466187, 0.21113380789756775, -0.12497640401124954, 0.6221578121185303, 1.4068548679351807, -0.39673084020614624, 2.4263086318969727, 0.35435256361961365, -0.8835663795471191, 1.5816757678985596, -1.6411951780319214, -0.5146673917770386, 0.18971872329711914, -0.16779585182666779, -0.4214591979980469, -0.37413132190704346, 0.43514561653137207, 0.16819903254508972, 0.7814382314682007, -0.20421116054058075, -0.0038356047589331865, 0.11027687042951584, 0.6696363091468811, -1.4491320848464966, 0.19423310458660126, -0.05683278664946556, 0.7207713723182678, -0.9496418833732605, -1.0077705383300781, -0.5968700647354126, -0.028399981558322906, 0.7928248047828674, 0.22889883816242218, -1.1188466548919678, -0.6440182328224182, 0.6532660126686096, 1.4086302518844604, 0.07627470046281815, -0.8585742115974426, 0.20224063098430634, -0.5413571000099182, -1.2697359323501587, 1.0982208251953125, -0.14073413610458374, 0.28770068287849426, -1.7136257886886597, -0.40925538539886475, 0.6148024201393127, -0.4533270597457886, -0.23303698003292084, 2.0423386096954346, -1.0297281742095947, -0.8678867220878601, 0.2730708420276642, 0.9837522506713867, -0.4609626829624176, -1.020012378692627, 1.1570974588394165, 0.7619825005531311, 0.773176372051239, 0.6660469770431519, 1.6040023565292358, 0.6121339201927185, 0.66154944896698, -1.384795069694519, -0.1052161455154419, -0.0895981416106224, 0.6411675810813904, 0.7782564759254456, -0.016149653121829033, 0.1072121188044548, -0.5257972478866577, 1.2281590700149536, -0.7139047384262085, 0.15352687239646912, -2.476085662841797, 0.39201322197914124, -2.2697179317474365, -1.2280579805374146, 1.702622652053833, 0.8541083931922913, 1.0679980516433716, -2.8943536281585693, -1.8385509252548218, -0.3569878041744232, -1.1654114723205566, 0.09769392758607864, -0.359950989484787, -0.03257220238447189, 0.6892731785774231, -0.8928927779197693, 0.0515495128929615, -0.12567149102687836, 0.5875986218452454, 1.1954127550125122, 0.8814269304275513, 0.12713512778282166, 0.17103905975818634, 0.45150381326675415, -1.0424758195877075, -0.4762720763683319, -0.5711147785186768, -1.8262708187103271, -0.03632170706987381, 0.13296367228031158], [-0.4153098464012146, 0.08946952223777771, 1.141203761100769, -0.9468303322792053, -0.10679991543292999, -1.9196577072143555, 0.5874086618423462, -1.710819959640503, 0.30946967005729675, -0.8283117413520813, 1.4494190216064453, -0.0915389135479927, -0.6741134524345398, -0.36858972907066345, 0.044339943677186966, 0.3163285255432129, -0.7014951109886169, 0.05609341710805893, 0.5940307378768921, -1.0288612842559814, -0.2465711534023285, -1.3048782348632812, 0.24224890768527985, 0.21252161264419556, -0.9346742630004883, -1.0136164426803589, 0.8561862707138062, 0.3713926374912262, -0.05188295245170593, 0.7375897169113159, 0.09760770201683044, -1.0432809591293335, 0.0348876416683197, 1.3961505889892578, -1.030921459197998, 0.18663841485977173, 2.4954674243927, 1.0533565282821655, -0.6459572315216064, -1.1560205221176147, -0.563275158405304, -1.5739145278930664, 2.362320899963379, 0.6605736613273621, -0.36909812688827515, -1.2708613872528076, -0.024618366733193398, -0.5797237157821655, -0.8028209209442139, 0.3620671331882477, -0.22011451423168182, -1.2760854959487915, 0.9242066144943237, 1.1175975799560547, -0.06269896030426025, 0.9733220934867859, -0.05630692094564438, -1.0721873044967651, 0.7336218357086182, 0.15369287133216858, 0.37396934628486633, -2.254624843597412, 0.5860451459884644, -0.6712955236434937, -0.6114338636398315, 0.6040242314338684, 0.7712996006011963, 0.5036115646362305, -0.13944241404533386, -2.079420804977417, 0.23437125980854034, -0.06468106806278229, 1.3210610151290894, -0.4306166470050812, -1.7403002977371216, -2.3168785572052, -0.5776168704032898, 0.48426127433776855, 0.524503767490387, -2.1694507598876953, 0.6453719735145569, 0.12659089267253876, -0.8453257083892822, 2.3659520149230957, 1.474977970123291, 0.2934530973434448, -1.4519129991531372, 0.41530081629753113, 1.5613945722579956, -2.396268606185913, 0.027414187788963318, -1.3291127681732178, -0.11126727610826492, -1.262008786201477, 1.0990430116653442, 0.4404567778110504, -0.35755112767219543, -0.2030106782913208, -0.24196307361125946, -0.8442224860191345, 0.43780916929244995, -0.5485925078392029, -0.9971822500228882, 1.5210936069488525, -0.05918744578957558, -1.5870033502578735, -0.3714173436164856, -0.34622785449028015, -0.7632506489753723, 0.9256674647331238, -3.676565408706665, -0.371568500995636, -0.6763830780982971, 0.990644633769989, -1.1390656232833862, 0.3932783007621765, -0.7677229046821594, 1.1898335218429565, -0.09427987039089203, -0.905756413936615, -1.16669499874115, 2.554708957672119, -0.5720217227935791, -1.0504499673843384, -1.2630597352981567, -1.7610726356506348, 0.6518604159355164, 0.8618940114974976], [0.4838707149028778, -1.1042898893356323, -0.4392187297344208, -0.22098438441753387, -2.025383472442627, 0.4938288927078247, 0.011347406543791294, 0.9640204906463623, -0.14627625048160553, 0.7303602695465088, 1.6557177305221558, -0.5297216176986694, -0.4239386022090912, -1.5359431505203247, 0.43121036887168884, 0.5015302300453186, 0.03469874709844589, -0.7643928527832031, -0.28828585147857666, -1.5619415044784546, -0.7624968886375427, 0.6991309523582458, -1.9067630767822266, -0.6386339664459229, -1.0279754400253296, 0.5677012205123901, 0.4531717896461487, -0.07167200744152069, 0.12081008404493332, 0.017606012523174286, -0.20649760961532593, 1.1685518026351929, 0.5785908102989197, -0.7157228589057922, 0.002744122641161084, 1.2629621028900146, -1.5024304389953613, 0.47704240679740906, 0.3032848834991455, -0.3830394148826599, -0.11759689450263977, 1.2533122301101685, -0.07685340940952301, 0.6544131636619568, 1.1124285459518433, -0.041473422199487686, -0.05794882774353027, 0.024079829454421997, -0.7157328724861145, -0.06200899928808212, -0.5986896753311157, -0.3302421271800995, -0.6685652136802673, -0.10667908936738968, 1.465835690498352, 0.8556123375892639, -1.3740308284759521, 0.06115978583693504, -1.187727928161621, 0.5546932816505432, -0.011014890857040882, 0.850555956363678, -0.22915621101856232, -0.8363286256790161, 1.4404807090759277, -0.7521394491195679, 1.908991813659668, 0.14965660870075226, -0.16585668921470642, 0.47308555245399475, 0.6392795443534851, -0.8117702007293701, 0.19341260194778442, -0.21191652119159698, -0.5026574730873108, 0.554283618927002, -0.6483007669448853, 0.6196282505989075, -1.2168465852737427, -0.4181704521179199, -0.9160300493240356, 2.7365994453430176, 0.07971161603927612, -0.922635555267334, 1.3997306823730469, -0.07694894075393677, -1.5640065670013428, -0.7586762309074402, 0.09976179897785187, 0.5594790577888489, 0.18757960200309753, 3.102139711380005, 0.2665037214756012, -0.3388833999633789, -1.5094189643859863, 0.14706115424633026, 0.04873969405889511, -0.16919374465942383, -1.0447300672531128, 0.4793207347393036, -1.3967952728271484, -1.4328302145004272, -1.7032057046890259, -1.4804768562316895, 0.962732195854187, -2.466576099395752, 0.20984862744808197, 0.01089952327311039, -1.5782709121704102, -1.0464396476745605, -0.9638025760650635, -1.9422551393508911, 1.1872881650924683, 2.1194872856140137, -0.44680169224739075, 0.44824379682540894, -1.4453132152557373, -0.4147922396659851, 1.9329084157943726, -0.3092098534107208, 0.09129222482442856, 0.1316412091255188, 2.4045302867889404, -0.2486071139574051, -0.803092896938324, 0.24895039200782776, -1.1086516380310059, -0.8532854914665222], [-0.4868137538433075, -0.17013250291347504, -0.16765454411506653, 0.10430783033370972, -0.7082282304763794, 0.3998446464538574, 1.7744498252868652, 0.7540920376777649, 0.19107207655906677, -0.6860674023628235, 1.340855598449707, 0.46605485677719116, 0.03892204910516739, 1.70653235912323, -0.13738302886486053, -1.2604742050170898, -0.739965558052063, 0.3897268772125244, -1.0493651628494263, 0.6590180397033691, -0.897473931312561, 2.0229198932647705, 0.4590993821620941, -1.3089849948883057, 0.16726288199424744, -0.705479085445404, 0.4501813054084778, 1.2375414371490479, -0.9648573994636536, -2.196568250656128, -0.1488284468650818, 0.10088122636079788, 1.4690451622009277, 0.7844692468643188, -0.17955918610095978, -0.0761025920510292, 0.5161503553390503, 1.5630840063095093, 0.41234466433525085, -0.2327861487865448, -0.5152439475059509, 0.9903426170349121, 1.485689640045166, -1.2608331441879272, -0.509309709072113, 0.052519217133522034, 0.39222195744514465, -0.16126728057861328, 2.261458158493042, 0.20625749230384827, -2.4295084476470947, -0.15335366129875183, 1.7660574913024902, -1.2991502285003662, -0.0448555126786232, 1.8191477060317993, 0.4929179847240448, -0.48910537362098694, 1.3937100172042847, 0.2649461328983307, -0.031828779727220535, 0.877677857875824, -0.2163219451904297, -1.103721261024475, 0.6737138628959656, -0.21321609616279602, -1.2183083295822144, -0.5594040155410767, 0.23947857320308685, -2.030251979827881, -0.21245695650577545, -0.9607701301574707, 0.2605515122413635, 1.3265295028686523, -1.2645716667175293, 0.9316990375518799, 0.39613571763038635, -0.13099981844425201, -1.1649959087371826, 0.8529151082038879, 0.8332352042198181, 0.6490631103515625, -1.3677717447280884, -1.945617914199829, 0.7377487421035767, -0.5289825797080994, 0.7656707763671875, -1.247709035873413, -0.05720097944140434, 1.897933006286621, 0.4967271685600281, 0.7972607612609863, 0.2053034007549286, 0.32766208052635193, 0.582629382610321, -0.3715263903141022, 0.6460086107254028, 2.090198040008545, -0.7640935778617859, 1.3283265829086304, 1.6708780527114868, -1.0326430797576904, 0.24000734090805054, 1.8620949983596802, 0.3124385476112366, 1.4095195531845093, -0.6159989237785339, 0.34329041838645935, -0.4279715418815613, 0.15484702587127686, -0.011417508125305176, -0.8798989057540894, 0.5579429864883423, -0.13487015664577484, -1.101911187171936, -0.981955885887146, -0.5994408130645752, 0.9546995759010315, 0.15186648070812225, 0.8290058970451355, 0.09077631682157516, -0.011748891323804855, 0.766736626625061, -0.3660060465335846, -0.09212809056043625, 0.07611603289842606, -1.6886464357376099, -0.5987695455551147], [-1.4896564483642578, -0.24167852103710175, -0.6500977873802185, 0.4570785164833069, 0.9731053113937378, 1.8077785968780518, 0.38475939631462097, -0.6857045292854309, -0.06212081387639046, -1.493507742881775, 0.24934101104736328, 2.1320600509643555, 0.0027522945310920477, 0.3782846927642822, -1.3772387504577637, -0.26058539748191833, -0.24510276317596436, -0.03200672194361687, -0.2146691083908081, 0.18574360013008118, -1.2719769477844238, -1.1865556240081787, 0.590042233467102, 0.8652328848838806, -1.4157521724700928, -0.00258127530105412, 0.018608709797263145, 0.7294019460678101, -0.2625880539417267, -2.3920931816101074, 0.4140913784503937, -0.4961059093475342, -2.138002634048462, 0.9031932950019836, 1.9273381233215332, -0.28376248478889465, -0.10463371872901917, -0.3330594301223755, -0.2552242577075958, 0.035352081060409546, 1.38712477684021, 0.9344351887702942, -0.8070129752159119, -1.8537226915359497, -1.4410687685012817, -1.9848295450210571, -0.15280234813690186, 0.21759356558322906, 1.5990979671478271, -0.012852574698626995, 1.2059794664382935, 0.7192323207855225, 0.31491991877555847, 2.1279730796813965, 0.11082891374826431, -0.07496507465839386, 1.8006656169891357, -0.2650228440761566, 0.6650441884994507, -1.4385051727294922, -0.5525602698326111, -0.6974650025367737, 0.541187047958374, 0.94908207654953, 0.41629815101623535, 0.8716964721679688, 2.5373330116271973, 1.303526520729065, -0.2117832750082016, 0.8771044611930847, -1.3636276721954346, -1.0784761905670166, 0.22671152651309967, 0.33037588000297546, -0.998534083366394, 2.498863697052002, -0.5442246794700623, 0.9657295346260071, 0.20770792663097382, 0.6112193465232849, 1.1922038793563843, 2.406006097793579, -0.556727409362793, 0.4251355528831482, 0.24028946459293365, 0.6851707100868225, -1.6808916330337524, 1.8668458461761475, 0.0798652172088623, 0.5736140608787537, 0.4385422170162201, 1.334212303161621, -0.16239787638187408, 0.4232698678970337, 0.9074589610099792, -0.0644131675362587, 0.7816205620765686, 2.507678270339966, -0.3274681866168976, 0.4250938892364502, 2.135007858276367, 0.5285333395004272, -1.3423597812652588, 0.2979462742805481, -0.05666319653391838, 0.6511585116386414, -0.37503188848495483, -0.3854202628135681, 0.32745835185050964, -1.2219972610473633, -1.5778143405914307, 0.47406280040740967, 1.524743914604187, 0.8358551859855652, -0.4986169934272766, -0.6229773163795471, 1.6575672626495361, -0.17081816494464874, 0.2590298652648926, 0.6769309639930725, 0.69100022315979, -1.4076346158981323, 0.004784812219440937, -0.05469220504164696, -0.14969147741794586, 0.67539381980896, -1.3600668907165527, 0.2112027406692505], [0.16928695142269135, -2.4831767082214355, -1.6839678287506104, 0.6447790861129761, 0.2451929748058319, -2.2207579612731934, 0.005509461276233196, 1.2699530124664307, 1.3693844079971313, 0.16927874088287354, 0.10013128072023392, 0.4028508961200714, 0.4911675453186035, -0.9677478671073914, -1.0621329545974731, 1.1687016487121582, 0.24551288783550262, 0.982064962387085, -2.406430244445801, 0.8129143714904785, -0.4343697428703308, -0.4707954525947571, 1.3669469356536865, -0.3949050009250641, 1.5194299221038818, 2.669506788253784, 0.9461994171142578, 0.8916197419166565, 0.07882486283779144, 0.2658299207687378, -0.11792658269405365, -0.9413959383964539, -0.5667180418968201, 0.33652982115745544, -0.5807943344116211, 0.7982211709022522, 0.5083645582199097, 0.03993714600801468, 0.04203220084309578, -0.38697272539138794, 0.685691237449646, -1.4652599096298218, -0.41046586632728577, -2.109180450439453, -0.029827263206243515, 0.3371126353740692, -0.00024412028142251074, 0.3012763261795044, 0.7403980493545532, 0.8735666275024414, 1.5585013628005981, 0.2758813202381134, -0.03856789693236351, 0.11033208668231964, 0.6020841002464294, 0.2084202617406845, 0.6457250118255615, 2.027616262435913, 0.19956767559051514, 2.390913963317871, -0.5480151176452637, -0.9143637418746948, 0.40658941864967346, -1.345799207687378, -2.428725004196167, -0.5413163304328918, -0.23559893667697906, -0.17913208901882172, 0.2910107970237732, 0.42313772439956665, -1.3763272762298584, -0.5392293334007263, -0.878448486328125, 0.5081519484519958, -0.4053417146205902, 0.5720625519752502, 1.2342579364776611, 1.4523909091949463, -0.06674370169639587, 1.1352014541625977, -2.620083808898926, 0.4405961334705353, -1.1105332374572754, -0.3928743600845337, 0.953525722026825, 1.1448556184768677, -1.6744648218154907, -0.43665096163749695, -0.1976323276758194, 1.3667724132537842, 0.7833513617515564, 1.6240627765655518, 0.5189031362533569, -2.4923605918884277, 0.06964050978422165, 0.7808114290237427, -0.9832247495651245, -0.6710816025733948, -0.5811695456504822, 1.422752857208252, -0.5804827809333801, -1.3281776905059814, 0.29098162055015564, -0.0035569295287132263, -0.9941650032997131, -0.052949510514736176, -0.5479521751403809, 1.1624804735183716, -0.6791669130325317, 2.2772204875946045, 1.8770852088928223, 1.8116568326950073, -0.9383693337440491, -0.9974289536476135, -0.5912306904792786, 0.5776275992393494, -1.197838544845581, 0.9535843133926392, -0.7569620609283447, 1.1969023942947388, -2.270413398742676, 2.3433353900909424, -0.4441203773021698, 0.3621537387371063, -0.43078917264938354, -0.10332565754652023, 1.1317322254180908, 0.15963031351566315], [0.5453232526779175, -1.815483570098877, -0.6960895657539368, 1.567298412322998, -0.07243777066469193, -0.54636150598526, -0.18639425933361053, -0.11913835257291794, -1.3939199447631836, -0.6926465034484863, 0.33607178926467896, 0.16425803303718567, 0.32249531149864197, -0.35623180866241455, 0.15177206695079803, 0.7042546272277832, -0.7611901760101318, -1.2432279586791992, 0.20293311774730682, -1.4418182373046875, 1.3491699695587158, 0.6007348299026489, 0.9364913105964661, -0.6906372308731079, -0.5753207206726074, 0.13724252581596375, 1.1918203830718994, 0.12751628458499908, 0.17650392651557922, 0.25958123803138733, 2.9758493900299072, -0.16145870089530945, -0.32503753900527954, -1.641356348991394, 0.5941392183303833, 0.24220681190490723, -2.7585630416870117, 0.0993446335196495, -1.812612533569336, 0.04319421201944351, -1.1247782707214355, -0.9039769768714905, 0.149897500872612, -0.3972640037536621, 0.3442118465900421, 1.3162809610366821, 0.8400353193283081, 0.05712065100669861, 1.1686955690383911, 0.3792717158794403, 0.10784734785556793, 0.10444952547550201, 0.7321494817733765, 0.19893993437290192, -0.2264672815799713, -0.09757999330759048, 0.8350509405136108, 2.368689775466919, -0.10246913880109787, 0.6360069513320923, 1.4295597076416016, -0.245492622256279, 0.3334025740623474, 0.4362340569496155, 0.5792438983917236, -0.5269665122032166, 1.5381758213043213, -1.0165610313415527, -1.1755225658416748, -1.1987779140472412, -0.12972503900527954, 0.5297259092330933, -0.9463954567909241, 1.6235733032226562, -0.04428669810295105, -0.008892270736396313, -0.5788196325302124, -1.6116135120391846, 1.0283983945846558, 1.6812182664871216, -1.0055909156799316, -1.1661567687988281, 2.53592848777771, -2.0929548740386963, 1.4603079557418823, 1.114245057106018, 0.0011718407040461898, 1.7372353076934814, -1.0050584077835083, -0.19859609007835388, 1.1397693157196045, -0.7205429077148438, -0.8166181445121765, 0.14181700348854065, -1.6643991470336914, -0.7282513976097107, 0.27073460817337036, -0.8856858015060425, 1.7453043460845947, -0.8163242340087891, -0.6662157773971558, 0.29140201210975647, 1.329397201538086, -1.3280212879180908, 0.07096979767084122, -1.5046762228012085, -1.212929606437683, 0.08298035711050034, 1.1710247993469238, -1.1673723459243774, -0.21866612136363983, 0.8597528338432312, -1.6009416580200195, 0.2416907250881195, 0.7299624085426331, -0.9711588621139526, -1.4241255521774292, 1.2362430095672607, -0.047507479786872864, -0.14778463542461395, 1.8282524347305298, 0.298576682806015, 0.5100271701812744, 1.3191289901733398, -0.9994471669197083, 1.0613797903060913, 2.1452457904815674, -1.1721776723861694], [-0.8261048793792725, 0.3081543743610382, 1.0644158124923706, 0.2607176601886749, -0.6272707581520081, 1.533403754234314, 1.0432568788528442, 2.0701754093170166, 0.3525901436805725, 0.4317319095134735, 0.004361970815807581, -1.8550113439559937, 0.7659482359886169, 0.5868254899978638, -1.4444533586502075, -1.497053623199463, -0.6451149582862854, 0.6993755102157593, -0.10982612520456314, -1.1695767641067505, 0.5788390636444092, 1.0954248905181885, 0.28992345929145813, -0.8094128966331482, 1.0903061628341675, 0.8441647291183472, -0.5795580148696899, 0.773917019367218, -1.3796919584274292, -1.4971600770950317, -1.391982913017273, 0.5055978298187256, 1.3549315929412842, -0.815515398979187, 0.21140718460083008, 1.0615730285644531, 0.23757094144821167, 1.3486305475234985, 1.3807905912399292, 0.14996084570884705, 0.08830683678388596, 1.5151439905166626, -0.5195848941802979, -0.5589275360107422, -0.2890275716781616, 2.7855846881866455, -1.2815998792648315, -0.8119180798530579, -0.9974632859230042, 0.9158757925033569, -0.4344393014907837, 1.0461808443069458, 0.37641751766204834, -2.132146120071411, -1.5105066299438477, 0.12659160792827606, -0.31700342893600464, 0.1596105843782425, -1.657344937324524, 0.44999659061431885, 0.5451391935348511, -0.4847927391529083, 0.6259570717811584, 0.32842063903808594, -0.2898007333278656, -0.39248478412628174, 1.55245840549469, -0.20193926990032196, 1.3633298873901367, -0.4440693259239197, -0.5353456735610962, -0.4265732765197754, -1.327116847038269, 1.289608359336853, -0.5769439935684204, -0.41692233085632324, 1.6368635892868042, -0.018631137907505035, 0.29426926374435425, 0.7114923000335693, 1.2222096920013428, -0.7865070104598999, -0.7094928622245789, 0.7789052724838257, 0.5509126782417297, 1.0937812328338623, 0.7482947707176208, 0.8945977687835693, -0.2500017583370209, 1.6919678449630737, 0.4630855917930603, 0.7167940139770508, 2.4200587272644043, -0.7108697295188904, -0.4207831025123596, 1.0076297521591187, 1.4909884929656982, -0.8877121210098267, 0.06671813130378723, -0.2237740159034729, -0.3716481029987335, -1.6688828468322754, 0.11327744275331497, 0.9903875589370728, -0.25016042590141296, -0.07268427312374115, 2.006704807281494, -0.15134580433368683, -0.14268681406974792, -0.8608483076095581, -0.24162274599075317, 0.8243441581726074, 0.21259629726409912, -0.3302500247955322, 2.5378549098968506, 0.4423374533653259, 0.9888971447944641, -0.09573046863079071, -0.17425674200057983, 1.2201876640319824, -0.33915090560913086, -2.2083733081817627, -0.08592752367258072, 0.35037267208099365, 0.11935645341873169, -0.039544958621263504, 0.2815566658973694, 0.8254660964012146], [-0.833075761795044, -0.9597572684288025, -0.6071398258209229, 0.3166397511959076, 0.9266406893730164, 0.4372674226760864, 0.34028133749961853, -1.2674320936203003, -0.6649677157402039, 0.2252984195947647, 0.27135390043258667, 1.8986873626708984, -0.7194719314575195, 1.0821505784988403, 0.6276437640190125, 1.2223843336105347, -1.4351216554641724, -0.3201742172241211, 0.5398643612861633, 0.1989317536354065, 0.6319535970687866, -0.36370840668678284, 0.048596516251564026, 0.22865243256092072, -1.0708004236221313, -0.9960131049156189, 0.8498736023902893, 0.036208923906087875, -1.4834591150283813, 1.5795480012893677, -0.6627835035324097, -0.6885837912559509, 1.9557040929794312, -0.28328776359558105, -1.3452776670455933, 1.19999098777771, -0.10600363463163376, -1.403353214263916, 0.24476245045661926, -0.15829747915267944, 2.9617698192596436, 0.6878650188446045, 0.7113451361656189, -0.5399623513221741, 1.066968321800232, 0.8130788207054138, 2.3521947860717773, -0.6062409281730652, -0.9251431226730347, -0.9783487915992737, -0.7190587520599365, 1.8858263492584229, 0.7962130904197693, -1.1013885736465454, 0.7048488259315491, -0.18949730694293976, 1.324371337890625, -0.23426453769207, -0.27376219630241394, 0.2968461215496063, 0.15578752756118774, 0.39790189266204834, 0.8814275860786438, 1.4708592891693115, -0.831453263759613, -0.49356135725975037, 0.7149316668510437, 0.5634313225746155, -0.7268074750900269, -2.0986828804016113, 1.4396227598190308, 0.32466447353363037, 0.7721675634384155, -1.5665642023086548, 3.1490259170532227, -0.6861559152603149, -0.7259999513626099, -0.10127155482769012, 1.8967682123184204, -0.5460783243179321, -1.4322417974472046, 0.294635146856308, 1.584219217300415, 1.4398916959762573, 0.43710657954216003, 2.1508452892303467, -0.7547243237495422, 0.1716158092021942, -1.09048593044281, -1.7022678852081299, 3.181063652038574, -0.1116708517074585, 0.7775635719299316, 0.18382200598716736, 0.8986465930938721, 0.9520323872566223, 0.5932614207267761, -1.0248522758483887, -0.9995847940444946, -0.10702010989189148, 1.64812433719635, 0.18365907669067383, -0.3850044012069702, 1.1292723417282104, 0.21093899011611938, 0.6978432536125183, -0.19539690017700195, 0.4576154947280884, 0.17364948987960815, -0.550942599773407, 1.1139742136001587, -0.09507998079061508, 0.2682957351207733, -1.9024053812026978, 0.05426062270998955, -0.514556348323822, -0.18052929639816284, -2.470477342605591, 1.073699712753296, -0.741704523563385, 1.2275738716125488, 1.2286934852600098, 0.8425689935684204, -0.17337748408317566, -0.10788343101739883, 0.8381758332252502, 0.9274640679359436, -0.555582582950592], [-0.42621710896492004, -0.11989984661340714, -0.8845435380935669, 1.0333677530288696, 1.0884898900985718, 0.14426235854625702, 0.1617957204580307, -0.23678922653198242, 1.4771525859832764, -1.5944879055023193, -1.3509780168533325, -0.7915992736816406, 0.5105772614479065, -0.10874484479427338, -0.4361982047557831, -2.0273404121398926, -0.9837099313735962, 1.0030839443206787, -1.4077235460281372, -0.09682139754295349, 0.4984305799007416, 0.5187281370162964, 2.1711349487304688, 1.3529915809631348, -1.6721537113189697, 0.1700575053691864, -0.5163642764091492, -0.736849308013916, -0.1851383000612259, -1.7586801052093506, 0.0566815584897995, 2.1953985691070557, -0.2557881474494934, -0.929825484752655, -1.198227047920227, 3.089674711227417, -2.401995897293091, -0.48952749371528625, -0.8626078367233276, -1.1646100282669067, 0.6589593291282654, 0.2257784754037857, 0.8497634530067444, -0.5703262090682983, -0.29289060831069946, 0.29976916313171387, 1.155085802078247, -0.874323844909668, 0.06729188561439514, 0.13619090616703033, -0.016257086768746376, 0.06605371832847595, -0.5537993907928467, 1.1183531284332275, -0.76798015832901, -0.6948880553245544, 0.3115396797657013, 0.26350539922714233, 0.2407289296388626, -0.7263760566711426, -0.5696288347244263, -0.09935420751571655, -0.8323805332183838, 0.014258656650781631, 0.2759643793106079, -1.5694019794464111, 1.6102118492126465, 0.7686373591423035, -1.807058334350586, 0.4230731129646301, -2.300708532333374, 2.036957025527954, 0.9077573418617249, -1.8425571918487549, -0.11934466660022736, 0.7637050747871399, -0.29812613129615784, 1.2514394521713257, -1.0705866813659668, 0.682604193687439, -0.5443567633628845, -1.6049563884735107, -1.1403253078460693, -0.4876902401447296, 0.5217891931533813, -1.3257302045822144, 1.347032904624939, 2.0464420318603516, 0.28241533041000366, 1.638061761856079, -0.2333202064037323, -0.030522046610713005, 1.7460335493087769, 0.12004824727773666, 0.9975029230117798, -0.9306139945983887, 0.3477135896682739, 0.49862807989120483, -0.5596521496772766, 0.1453174650669098, 0.11446237564086914, 0.232639878988266, 0.12392479181289673, 0.4259573221206665, -1.4092624187469482, 0.09359102696180344, -0.39138439297676086, -0.36670684814453125, 0.42758315801620483, 2.1027488708496094, 0.10142973810434341, -0.43207263946533203, 0.1016250029206276, -0.005574893206357956, -0.2108258605003357, -1.3447341918945312, -0.267399400472641, -0.20713143050670624, 0.044452980160713196, 0.126173198223114, 1.2327640056610107, -0.9121478796005249, -0.60736083984375, -0.45141714811325073, 0.7279302477836609, 0.2815588116645813, -1.5312503576278687, -0.35409238934516907], [-0.06614743918180466, 0.2281375378370285, -0.7671670317649841, -0.44766291975975037, 0.6537735462188721, -0.044102832674980164, 1.8848427534103394, 1.821044921875, -1.0467602014541626, 0.0291171632707119, -0.11445143818855286, -0.4519534111022949, 0.3122207820415497, -1.1825971603393555, -0.631406843662262, -0.21582333743572235, 0.2723294496536255, -0.04153002053499222, -0.1451873630285263, 0.9113369584083557, -1.1596966981887817, 0.556545078754425, -0.9698084592819214, 0.3203960657119751, -0.5903647541999817, 1.0489193201065063, 0.010055556893348694, 0.5790379643440247, 0.6907095313072205, -0.4348679184913635, 0.6517065167427063, 0.9940676689147949, 0.6374779343605042, 1.0575438737869263, -0.12227597832679749, 1.116502285003662, -0.5607845187187195, -0.07911407947540283, 0.7104536294937134, 0.3985942006111145, -0.7508053183555603, -0.18726274371147156, -0.8924145698547363, 0.0039453632198274136, -0.22464993596076965, -0.6330054402351379, -0.19645139575004578, -0.20820602774620056, -0.8457380533218384, 0.43213626742362976, 0.04215698316693306, -0.36532023549079895, -0.19174106419086456, 0.3535161018371582, -0.2566734254360199, -0.3980283737182617, 0.5635385513305664, -0.24762384593486786, -0.39011380076408386, 1.7348822355270386, 1.6318753957748413, 1.0561044216156006, -0.00795541238039732, -0.9065543413162231, 0.4491264224052429, -0.41414329409599304, 0.5193573236465454, 2.010279417037964, 0.7664074897766113, -1.6534538269042969, 1.1617050170898438, -1.2978445291519165, -0.28261345624923706, -0.9818887710571289, -1.1802723407745361, -0.02493569813668728, -0.9641836881637573, -0.794969916343689, 0.2505498230457306, 0.03306414186954498, -0.6222915649414062, -1.4688690900802612, 1.491924524307251, 1.4303371906280518, -0.49658140540122986, 0.027450909838080406, 0.41538307070732117, 1.2147612571716309, 0.6603150367736816, -1.0308654308319092, 0.034155189990997314, -1.5016132593154907, -0.009636186994612217, 0.717025637626648, 0.7611415386199951, 1.0836951732635498, 1.2002043724060059, -0.45549431443214417, -0.8245460391044617, 1.4451733827590942, -1.5849483013153076, 1.2178237438201904, 0.08525747805833817, 0.7993519306182861, -0.48035871982574463, -1.232784390449524, -0.547059953212738, -1.3745777606964111, -0.010491381399333477, 0.13615548610687256, 0.5983606576919556, 0.26674798130989075, 0.08860129863023758, -0.5301012992858887, -0.5237334370613098, -1.1644076108932495, 0.42021071910858154, -1.0675837993621826, -0.26852378249168396, -0.5358497500419617, -0.004489891696721315, -0.13028690218925476, 0.5642378926277161, -0.4879302382469177, 1.1239657402038574, 0.6740725040435791, 0.31867724657058716, -0.17648105323314667], [1.7309659719467163, 0.5318702459335327, -1.1432127952575684, -1.0955684185028076, -0.6170070171356201, 0.2890474200248718, 1.2004177570343018, 0.39202460646629333, 0.8268871903419495, -1.0319916009902954, -1.0185109376907349, 0.37255266308784485, 2.2450315952301025, -1.435699701309204, -0.3082905113697052, -0.9825969338417053, 0.8656454086303711, -1.5120456218719482, -0.0009679130744189024, 0.5095992088317871, 0.326600581407547, 0.512220025062561, -0.1313977986574173, 1.4525034427642822, -0.1799912303686142, 0.6723796129226685, -0.542682409286499, 0.9615965485572815, 0.037754226475954056, -0.45535069704055786, -1.7431412935256958, -0.6973673105239868, -0.920278787612915, 1.238200068473816, 1.467488169670105, -0.26742076873779297, 0.3795206546783447, 0.25479722023010254, 0.23853546380996704, -0.0641709715127945, 0.4327099919319153, -0.42660412192344666, 0.3184133768081665, 0.027745593339204788, 2.089818000793457, 0.27396729588508606, 0.2581576704978943, -0.846011221408844, -0.3959866464138031, 0.222430020570755, -0.7173979878425598, -0.3266722559928894, -1.2112023830413818, 1.2452176809310913, -1.8328436613082886, -2.1466102600097656, 1.0530695915222168, 0.6070660352706909, -1.6619714498519897, -2.1204726696014404, 0.5775477290153503, 0.8521772623062134, 1.3787962198257446, -0.8534472584724426, -0.10190923511981964, 0.8707864880561829, 2.2515978813171387, -2.008175849914551, 0.8921627998352051, 2.3361167907714844, 0.26871371269226074, -0.28064385056495667, 0.7936705946922302, -0.8851034045219421, 1.7925512790679932, 0.26717010140419006, 2.37420916557312, 0.2236345112323761, -1.6437768936157227, -0.682161808013916, -0.6972464323043823, -0.41019555926322937, -1.3549851179122925, -0.5866924524307251, -0.42692112922668457, 0.18257929384708405, -0.598311185836792, -0.22386519610881805, 0.8333139419555664, 0.7936558723449707, 0.2772519290447235, -1.5888813734054565, -0.11519303172826767, -0.2724185883998871, 0.44877657294273376, 1.4521416425704956, -0.891485869884491, -0.09494350105524063, -0.4768766164779663, 1.566066861152649, 0.17668145895004272, 0.8788750171661377, 0.5684989094734192, -1.0559333562850952, 0.2622734308242798, -0.9700539708137512, 1.8055446147918701, -1.0151879787445068, -0.2046935111284256, 0.407807320356369, -0.7658882737159729, 0.7434327602386475, 0.9520272612571716, -0.2917430102825165, -0.6995131373405457, 0.5645948648452759, 0.05235765501856804, -0.43385663628578186, -0.10062412917613983, -0.883303165435791, 2.136730194091797, 0.6968082189559937, -1.3611918687820435, 0.19002243876457214, -1.505765676498413, 1.0381237268447876, 0.4509339928627014, -0.8468934893608093], [-1.5356632471084595, 0.419216126203537, -1.1765097379684448, -0.13947342336177826, -1.4688876867294312, 0.6908449530601501, 0.4646095037460327, 0.23065288364887238, -0.4586654007434845, 0.35564082860946655, -1.5259394645690918, 0.4965287148952484, -0.04417532682418823, 0.3565312623977661, -1.3331154584884644, -1.0437906980514526, 1.489292860031128, 0.19427570700645447, 0.8571264743804932, -1.215408444404602, 0.4533463418483734, -0.6142604947090149, 0.2645748555660248, 0.4125073552131653, -1.5472387075424194, -1.383949875831604, 1.929018259048462, -0.1430703103542328, 2.064506769180298, 0.680357038974762, -0.3777473270893097, 0.03690283000469208, -0.759106457233429, -0.5473169684410095, -1.375909686088562, -0.7097340226173401, 0.8386027812957764, 0.724811315536499, -1.4098904132843018, -0.4631374776363373, -1.9835959672927856, 1.005367398262024, -0.902906596660614, 1.568917155265808, -0.43724608421325684, 0.12842677533626556, -0.6100980639457703, 0.10908353328704834, 0.9157769083976746, 0.7142431735992432, 0.19356007874011993, -0.5234003663063049, 0.6486959457397461, 0.49272847175598145, -0.7714799046516418, -0.41623052954673767, -0.146977499127388, 0.7308750748634338, 1.4155340194702148, -0.27097514271736145, 0.6126580834388733, 0.7449122667312622, 0.3293135166168213, -1.6645677089691162, -1.954380750656128, 0.551914632320404, 0.2833154797554016, 1.4997793436050415, 0.2724020779132843, -0.4885977506637573, 0.028025593608617783, -0.36229926347732544, 0.4818086326122284, 0.5138782858848572, -0.6068933010101318, -1.5223244428634644, 1.1107827425003052, 0.52436763048172, 1.2899630069732666, 0.6631059050559998, 1.9798121452331543, 0.9320042729377747, -0.37033119797706604, -1.2892038822174072, -0.16523678600788116, -0.40595850348472595, 1.2043789625167847, -0.5541825890541077, 0.07413787394762039, 0.39632079005241394, 0.2427094429731369, -0.09236208349466324, -0.13263678550720215, -0.8205909132957458, 0.7148134112358093, 2.067361831665039, 0.3750414550304413, 0.3019297122955322, 0.6139846444129944, -0.5959440469741821, -0.15718480944633484, -1.3685288429260254, 1.0467417240142822, -2.4041216373443604, 0.5302258133888245, -1.429772973060608, 0.8650093078613281, -0.7536674737930298, 0.8247542977333069, -0.3661664128303528, 0.973118245601654, 0.06116991862654686, 1.2620820999145508, -1.3064767122268677, -0.48888081312179565, 0.8402102589607239, -0.8127062320709229, -0.10808710008859634, -2.337977170944214, 0.9923623204231262, 0.6723804473876953, 1.0862853527069092, -0.1901673674583435, -1.2521976232528687, -1.1422398090362549, 1.0231809616088867, -0.31761837005615234, 0.9333420991897583], [-0.21242979168891907, 1.2532018423080444, -0.5971648693084717, -0.3880416750907898, 1.0478448867797852, -2.0833182334899902, 1.243178367614746, 1.1655492782592773, -0.39083144068717957, -1.3485249280929565, -0.3028686046600342, -1.0776139497756958, -0.34869998693466187, -0.9497783780097961, -0.8484141230583191, -1.0561860799789429, 0.30135276913642883, 0.3281545639038086, 1.452052354812622, 1.7788904905319214, 1.6495929956436157, -0.05594995245337486, -0.2661466598510742, -0.5179564356803894, 0.5591411590576172, 0.3842236399650574, -0.4979347288608551, -0.6695457696914673, 0.2703632414340973, 1.5367933511734009, -0.561821460723877, 0.8828921318054199, -1.148975133895874, 1.8416658639907837, 0.1901392638683319, 0.7586703896522522, 0.48638293147087097, 0.8936395645141602, 1.6040318012237549, 0.1937030702829361, 0.45577472448349, 1.0340758562088013, -0.19405411183834076, -1.2285919189453125, 0.3309481143951416, 1.047808289527893, 1.4275410175323486, 0.6275375485420227, -0.2548346519470215, -0.6628673076629639, 1.2403336763381958, -0.9686476588249207, 1.5030186176300049, 0.32703009247779846, -0.3028635084629059, -0.15235306322574615, -0.14010421931743622, 2.0397019386291504, -0.7421004772186279, -1.1082053184509277, -0.8219722509384155, 1.9305514097213745, -0.6128718852996826, -0.19888737797737122, 1.4018242359161377, 0.2759280800819397, -0.26312699913978577, -0.7396337389945984, -1.4245893955230713, -0.449148029088974, -1.0079610347747803, 2.315718173980713, 1.0415711402893066, 0.47189244627952576, 0.5941804051399231, -1.1044553518295288, -1.4116780757904053, -1.4524211883544922, -0.29896315932273865, -0.06997690349817276, 0.6379434466362, -2.0199925899505615, -0.9815109968185425, -0.013026205822825432, 0.4352165460586548, 0.3819110691547394, 0.9687578678131104, 0.10712286829948425, 0.5378357172012329, -0.023938000202178955, 1.2922393083572388, -0.0029188685584813356, 0.6352813243865967, -1.4137450456619263, -1.3000956773757935, -1.572144627571106, -0.34320691227912903, 1.20648193359375, 0.7991876602172852, -0.06398502737283707, 1.2208586931228638, 1.325313925743103, -0.728289783000946, -1.6099555492401123, 0.4503582715988159, 0.8513038158416748, 0.33354541659355164, -1.2367455959320068, 0.34841060638427734, -1.2596007585525513, 0.3968057334423065, -0.14919330179691315, -0.7880975604057312, 0.7944005727767944, -0.4379201829433441, 0.5573747158050537, 0.6074897050857544, 0.821292519569397, -1.6551445722579956, -0.052416834980249405, 1.2846755981445312, 0.8126636147499084, -1.2922953367233276, 0.9055734276771545, 1.1043936014175415, 0.042359016835689545, -0.13664887845516205, -0.5468165874481201]]], [[[-1.162335753440857, -0.7625601291656494, -0.44518420100212097, 0.25788745284080505, 0.9785229563713074, 0.10711058974266052, 0.5788000226020813, -1.724564790725708, -1.6180580854415894, -0.24970974028110504, -0.7962983846664429, -0.3039931058883667, 0.9713898301124573, -0.03990057855844498, 0.45520272850990295, 1.1545788049697876, -0.28996071219444275, 3.2143568992614746, -0.030070826411247253, -0.3350050151348114, -0.16119961440563202, 1.3276057243347168, 0.18545715510845184, -0.6780622005462646, 0.23199345171451569, 1.410524606704712, 0.4828125536441803, 0.6139604449272156, -0.4626713991165161, 0.18509997427463531, 0.39075765013694763, 0.10683474689722061, 0.22152239084243774, 1.4129022359848022, 0.22181011736392975, 1.404748797416687, 2.3492484092712402, 0.6647399663925171, 0.5819495320320129, 0.8535281419754028, -0.3707146644592285, -0.3021426200866699, 0.4872971177101135, -1.2899370193481445, -0.09438809752464294, 0.07924479991197586, -2.0367400646209717, 0.9885523319244385, 0.39479726552963257, -0.005380451213568449, 1.8257728815078735, -0.7865478992462158, -0.32081958651542664, 0.5216240286827087, -1.6808909177780151, -0.9816949963569641, -0.4366995692253113, 0.2603510320186615, -1.0680897235870361, -0.555016815662384, -0.2832033932209015, 1.494203805923462, 0.24532032012939453, 1.1918554306030273, -1.1243610382080078, -1.0421772003173828, 0.46032217144966125, -0.856275200843811, 1.3964152336120605, 0.5055970549583435, 0.14016889035701752, -0.7986996173858643, -1.868044376373291, -0.27816879749298096, 1.1656606197357178, -2.985987901687622, -1.3252867460250854, 0.18989995121955872, -0.2868749797344208, 0.6037196516990662, 0.11768889427185059, -0.19198383390903473, 0.9557780623435974, -0.6255386471748352, 0.7548840045928955, 1.4503145217895508, 0.4448806047439575, -0.0747564509510994, 1.0137832164764404, 1.6448256969451904, -1.9277254343032837, -0.6272504925727844, -0.07047280669212341, 0.9991206526756287, -1.1661810874938965, 0.4170234501361847, -0.45136967301368713, -1.09222412109375, -1.2888851165771484, 1.0990017652511597, -0.25183501839637756, -0.04266238212585449, 1.0948939323425293, 0.014871313236653805, 0.28055503964424133, 2.3037211894989014, 0.5135699510574341, -0.3846454918384552, -0.0635175108909607, -0.12389431893825531, 0.18652819097042084, 0.5546464323997498, 0.36188003420829773, 0.8386794328689575, -0.7090847492218018, -0.4790046513080597, 1.1447653770446777, -0.9373288750648499, -1.4875015020370483, 0.23254427313804626, -1.1507314443588257, -0.2553330659866333, -0.09053530544042587, -0.12511397898197174, 3.1813268661499023, -0.13227273523807526, 1.7314987182617188, 0.9113458395004272], [-0.5792399644851685, 0.43982023000717163, -1.4256335496902466, -0.7858002781867981, -0.32457542419433594, 0.09646844118833542, 1.1614097356796265, -0.04663202539086342, 1.5132331848144531, -0.04490461200475693, 1.150935173034668, 1.8563956022262573, -1.0155471563339233, -0.018881244584918022, -0.75264573097229, 0.03716159611940384, 0.06881238520145416, 0.6076029539108276, 0.7951083183288574, -1.3598076105117798, 0.3541429340839386, 2.1835336685180664, 1.1155062913894653, -1.5711952447891235, -0.8050141930580139, -1.0409555435180664, 2.5548787117004395, 0.9053448438644409, 0.807508647441864, -1.3743561506271362, -0.3730923533439636, 1.3569868803024292, 0.09476380795240402, 0.07898437231779099, 1.6841284036636353, 0.9680589437484741, -0.6934999227523804, 0.4847089648246765, 1.0054680109024048, 0.2416181117296219, 1.214666485786438, -0.282426655292511, -0.9617522358894348, -0.8871394991874695, -1.00406014919281, -1.3011902570724487, -1.9215412139892578, 1.6263822317123413, -0.16086134314537048, 0.06331454962491989, 0.955473005771637, -0.7998717427253723, 0.567406177520752, 0.21564117074012756, 0.5087541937828064, -0.8039599061012268, -0.6122598648071289, 0.6836554408073425, 0.8298212289810181, -0.2937726378440857, 0.00567359896376729, -0.1023927628993988, -0.3833211064338684, 0.6473978161811829, 0.05284890905022621, -0.5466273427009583, 0.10586980730295181, 1.731920838356018, 0.5214649438858032, -2.9537222385406494, -1.207558274269104, -0.3987632691860199, 0.25913727283477783, -0.8907320499420166, -1.4333168268203735, -0.11487632244825363, -0.7354603409767151, -0.1291790008544922, -1.1746361255645752, -0.2445596605539322, -1.3714070320129395, 1.2866419553756714, 0.41908523440361023, 0.44459161162376404, -0.738342821598053, -0.7303106784820557, 0.027694346383213997, 0.03693532571196556, -1.2670478820800781, -0.39250290393829346, 0.6036280393600464, 0.21749351918697357, 0.0907597467303276, -0.5382177829742432, -0.667370617389679, 0.7720798254013062, -0.6706127524375916, 1.0761830806732178, -0.36608925461769104, -0.342298299074173, -1.8796573877334595, 0.7366774082183838, 0.7417735457420349, 1.4533220529556274, 1.2575312852859497, 0.9594399333000183, 0.5831106305122375, -0.794193685054779, -1.0828831195831299, 0.11228562891483307, -1.279157042503357, -0.3064638674259186, -0.6182326078414917, 1.4090440273284912, 0.620854377746582, -1.4186381101608276, -0.7992009520530701, -0.5556820631027222, -2.752056121826172, 0.9242441058158875, 0.6463472247123718, 0.059742316603660583, -0.34196794033050537, -0.3569270670413971, -1.056551456451416, -1.0188928842544556, -0.15022428333759308, -0.3282089829444885], [-0.1357017606496811, -0.5652728080749512, -0.8482221364974976, 1.8101704120635986, 2.266057252883911, -0.27276545763015747, 0.49535849690437317, 0.4682067036628723, 0.3041183054447174, 0.24790458381175995, 1.3970040082931519, 1.8186137676239014, -1.3295063972473145, -0.06628106534481049, 1.8464821577072144, -1.950987696647644, -0.4156361520290375, -1.5579642057418823, -0.2828327715396881, -0.48094016313552856, -0.5020177364349365, 1.150139331817627, -0.5098968148231506, -0.7158947587013245, -0.7858006954193115, -0.3151891529560089, -0.17494919896125793, 1.0812879800796509, -1.2135553359985352, -0.006558041088283062, -1.2664713859558105, 0.09688909351825714, -0.06925825774669647, 0.9868759512901306, -0.36445051431655884, 0.8829445242881775, 0.38553136587142944, -0.25963419675827026, 1.320119023323059, 1.490410566329956, -1.3495457172393799, 1.4396864175796509, 1.2066882848739624, 0.4962984621524811, 0.35800468921661377, 1.2031713724136353, -0.2839956283569336, 0.2842823565006256, -0.8657957315444946, -1.0956428050994873, -1.8178000450134277, -0.7042976021766663, 0.4641708731651306, -1.210988998413086, 0.7969451546669006, -1.343341588973999, 0.4793429374694824, 1.6412967443466187, -0.44640567898750305, -0.5252412557601929, 0.38898125290870667, 1.6671276092529297, 1.3463822603225708, -0.07211197912693024, 0.6685739755630493, 0.11896375566720963, 0.3192330598831177, 0.0822453498840332, 1.1362674236297607, 0.0922638550400734, -0.5838987827301025, 1.5490694046020508, 0.4161888659000397, -0.9904984831809998, -1.0715316534042358, -1.7898449897766113, -0.14464986324310303, -2.2416253089904785, 0.7652804255485535, -0.11865539103746414, 1.1550458669662476, 0.77786785364151, 0.04188256338238716, -0.6095923781394958, 0.244369775056839, -0.489492267370224, 0.7912688851356506, 0.9349055886268616, 0.8471915125846863, -1.5851885080337524, 1.070434808731079, -0.08335857838392258, -0.011849197559058666, -0.6943589448928833, 0.018616287037730217, 0.10625966638326645, -0.6512053608894348, 0.038972239941358566, -0.1741388589143753, -1.3397672176361084, -0.10908111929893494, 0.8918304443359375, 0.6777260303497314, -0.15376998484134674, -1.361059546470642, -0.62491375207901, -0.36843690276145935, -0.8234022259712219, 0.27835017442703247, 0.7151347398757935, -0.2915152907371521, -0.5052916407585144, 1.2744908332824707, 0.15815915167331696, 0.9348865747451782, 0.17015717923641205, 0.10545774549245834, -0.3075115978717804, -2.2390029430389404, 0.2596120834350586, 2.380103349685669, 0.9927945733070374, 0.8750854134559631, 0.0885147824883461, 0.6678913831710815, -0.2370375096797943, -0.21929779648780823, -0.5645646452903748], [0.20757079124450684, -0.33016204833984375, 0.7553583383560181, -0.84609055519104, 1.2950760126113892, -0.29151350259780884, -0.8362318277359009, 0.3527323603630066, -1.2728334665298462, -0.24908329546451569, -0.2134721875190735, -1.9176255464553833, 0.11937406659126282, 1.4488717317581177, 0.9845037460327148, -2.562659740447998, -0.16995343565940857, -0.8725018501281738, 0.28490519523620605, 0.5986062288284302, -1.5584018230438232, -0.8098360896110535, 0.08711043745279312, -0.21413514018058777, 0.42507749795913696, -1.1372368335723877, 0.80157870054245, 1.0470678806304932, 1.2269748449325562, 1.578848958015442, -0.7852640748023987, 1.37481689453125, -0.150240957736969, 1.0675525665283203, -0.7763071060180664, 0.007970381528139114, -1.8075400590896606, 1.2443870306015015, 0.014415325596928596, 0.8731387257575989, -2.3516933917999268, -0.0964096412062645, 1.4110774993896484, -1.0859156847000122, 0.6947317123413086, 0.5144101977348328, -2.056299924850464, 1.3422768115997314, -1.0674848556518555, 0.4270056188106537, -3.355403423309326, -1.0860941410064697, -0.19619520008563995, 1.9082818031311035, 0.05470266938209534, -1.1523618698120117, -1.9290333986282349, 1.0855120420455933, -2.312190055847168, 1.5590318441390991, 0.8182725310325623, 1.612151026725769, -0.6964203715324402, 1.2295184135437012, -0.30915746092796326, -0.3462393879890442, 0.7949697971343994, 0.5900160074234009, 0.1394551545381546, -0.9446667432785034, 0.8502896428108215, 0.15370479226112366, -0.35417860746383667, -0.16648517549037933, -1.5975890159606934, 0.674075186252594, 0.6176983714103699, 0.04361710697412491, -2.263444185256958, -0.279395192861557, 0.21813592314720154, -0.4793519973754883, -1.4537792205810547, 0.23625019192695618, -2.411301851272583, -1.1723452806472778, -2.291177749633789, -0.20821942389011383, -0.04365839809179306, -0.42676693201065063, -0.2996987998485565, -0.0856749564409256, -1.5189666748046875, -0.39479824900627136, -1.114288568496704, 0.46280437707901, 0.08316896110773087, -0.18794800341129303, 1.211437702178955, 0.07255379855632782, -1.1924614906311035, 0.9424579739570618, 0.3922305703163147, 0.5598172545433044, -1.129465937614441, 1.220055103302002, 1.2563426494598389, -0.7164256572723389, 1.1270564794540405, 0.2984926998615265, 0.33783861994743347, 1.6308727264404297, -0.42062267661094666, 0.08488394320011139, -0.6553306579589844, -0.6896849274635315, 0.8094308376312256, -0.8016381859779358, -0.6607874035835266, -0.4793117046356201, 0.24241171777248383, -0.5724248886108398, -0.10470080375671387, 0.5350000262260437, -0.7917149662971497, -1.7895135879516602, 1.1016669273376465, -0.879416823387146], [0.35521745681762695, -1.4966952800750732, -0.5352216362953186, -1.4682589769363403, 1.6668131351470947, 0.7479786276817322, -0.08512585610151291, -1.398454189300537, -0.4354895353317261, -0.3832029700279236, 1.97464919090271, 0.5864238739013672, 0.034312207251787186, -1.585028886795044, 1.2891381978988647, -1.1480897665023804, -0.5263285636901855, -0.5970978140830994, 1.323735237121582, 0.2435319870710373, 0.049202121794223785, 1.774240493774414, -0.48574888706207275, 0.7632221579551697, -1.8949813842773438, 0.10268139839172363, 0.5777002573013306, -0.43965640664100647, 0.6779307126998901, 0.05189875513315201, -1.9278697967529297, -0.13699227571487427, 1.5205196142196655, 0.3145207464694977, 0.7113097906112671, 0.45564451813697815, 0.13611574470996857, 1.3901357650756836, -0.6652541160583496, -0.10038357228040695, 0.7188175916671753, 0.10367389768362045, -0.33932003378868103, 0.8039201498031616, -0.5181899666786194, 0.7199712991714478, 1.9763810634613037, 0.7963135838508606, -0.771904468536377, -0.6500137448310852, -0.5442289113998413, -1.459370493888855, -0.4944173991680145, 1.354427456855774, -0.08221409469842911, 1.731016993522644, 0.19597993791103363, -0.28447744250297546, -0.24513131380081177, 0.6278811693191528, 1.4245280027389526, -1.4089118242263794, -0.6997709274291992, -0.3583971858024597, 1.332289218902588, -0.455025315284729, 0.8027158379554749, -1.2020974159240723, -0.21227127313613892, 0.5231271982192993, 1.4522775411605835, -1.0336008071899414, 0.012360408902168274, -0.5300331115722656, -1.908075213432312, 1.5843678712844849, -0.1965618133544922, -0.990613579750061, 0.4684463441371918, -0.04410378634929657, 1.7272216081619263, 0.30934959650039673, -1.5661484003067017, 1.06610906124115, 1.4955955743789673, 0.1997898370027542, -0.9010801315307617, -0.22164420783519745, -0.5324512124061584, -1.8918076753616333, -0.7371467351913452, -0.5965982675552368, 0.3372596204280853, 0.030547672882676125, 1.7811528444290161, 0.5229504108428955, -0.5510923266410828, 2.4587762355804443, 0.24328286945819855, 1.1124955415725708, -0.7276006937026978, -0.6435457468032837, -0.2751210629940033, 0.2951093316078186, 1.0438063144683838, 1.8604786396026611, 1.0902831554412842, -0.03750206530094147, 0.5640076994895935, 0.6364309191703796, 0.027420828118920326, 0.06664639711380005, -1.16402006149292, 1.5751370191574097, 0.04493650794029236, 0.5298976302146912, -0.6368540525436401, 0.8120726346969604, 1.6511454582214355, -0.8653626441955566, -0.19543853402137756, 1.4659897089004517, 1.389534831047058, 1.6729822158813477, 0.4155603051185608, 0.4718964397907257, -0.1329742968082428, -0.8922855257987976], [-0.6664524674415588, 0.14382310211658478, 1.0922621488571167, -1.0858148336410522, -0.9097829461097717, 0.20643006265163422, 1.7049721479415894, -0.5351647138595581, 0.6902337074279785, -0.18944090604782104, 0.5929993391036987, -1.0979586839675903, 1.0753334760665894, 0.1224236860871315, -0.5541960000991821, -0.26819920539855957, -1.1074737310409546, -1.9526926279067993, 0.4156789183616638, 0.01829134300351143, 0.490193247795105, -1.170763611793518, -1.3804742097854614, -0.6375630497932434, 0.45379146933555603, -0.28380250930786133, 1.0622382164001465, 0.13405156135559082, 0.6764514446258545, 0.7301057577133179, -1.4639294147491455, 0.08816496282815933, -0.9300758838653564, 1.6215183734893799, -0.058141350746154785, -0.5207117199897766, 1.0154141187667847, -1.4411685466766357, 0.6394761800765991, 0.8048731088638306, -0.45397400856018066, 0.7436725497245789, 1.5232452154159546, -1.1825056076049805, -1.2843186855316162, 0.12504538893699646, -1.126607060432434, 1.0801736116409302, -0.5711143016815186, -0.5643048286437988, 0.7359011769294739, -1.0798057317733765, -0.5150655508041382, 1.5449373722076416, -0.5261061787605286, 0.6108396053314209, 0.25358277559280396, -1.2655783891677856, -0.2584128677845001, 0.12194730341434479, 0.9883841276168823, -1.006333351135254, -1.756556749343872, -0.22077856957912445, 0.22426898777484894, 0.8869081735610962, 0.5961045622825623, 0.8950862884521484, -0.2275511771440506, -0.00047387901577167213, -0.23600563406944275, 2.802821159362793, -0.4176822602748871, -0.35389792919158936, 0.15143722295761108, -0.3105001449584961, -0.5645835995674133, 0.29477939009666443, 0.2313946932554245, -0.3888164162635803, -0.1612681895494461, 0.7178561687469482, -2.193991184234619, -0.6827124953269958, -1.7300045490264893, 0.4196132719516754, -0.8247127532958984, -0.26866206526756287, 0.5026695132255554, 0.9042040705680847, -1.1928685903549194, 0.6243336796760559, 0.4015599489212036, -0.9496049880981445, 0.5754584074020386, -0.41600748896598816, -0.3401925265789032, -0.6253591179847717, -1.626138687133789, -0.4475739896297455, 1.04017174243927, -0.9989467859268188, -0.5627105236053467, -0.7405438423156738, 1.395790457725525, 0.8170791268348694, -0.13098302483558655, 0.13843677937984467, 1.4943535327911377, 0.09257703274488449, -1.0414717197418213, -1.4677715301513672, -0.1618949919939041, -2.2283530235290527, -0.6568993330001831, -0.6729313135147095, 1.6614336967468262, 1.1887742280960083, 1.6256799697875977, 0.2567744851112366, -0.3288753628730774, -0.09998679161071777, -0.8221825957298279, -1.8304611444473267, 1.6660785675048828, -0.5330352187156677, -1.8523424863815308, 1.4457157850265503], [1.9054968357086182, 0.12413979321718216, 0.1955835521221161, -1.428760051727295, -0.8333362936973572, -0.1318548023700714, -0.45660555362701416, -1.5148491859436035, -0.2065601497888565, 1.271804928779602, -0.4379494786262512, 0.8069830536842346, 0.34508004784584045, -0.34251052141189575, -2.0647802352905273, -0.1865817755460739, -1.1634469032287598, 0.38824373483657837, -0.4386087656021118, -1.0920497179031372, -1.6116926670074463, 0.6126755475997925, -0.11112818121910095, 0.3756486773490906, -2.5619189739227295, -1.2473363876342773, 1.5968401432037354, -1.2775225639343262, -0.03528537228703499, -1.3145123720169067, -0.29968494176864624, -0.08689973503351212, -0.45814085006713867, -0.4404234290122986, 0.26483267545700073, -1.194183349609375, -0.7066658735275269, 0.16334566473960876, 0.4257426857948303, 1.5231680870056152, -1.409101128578186, -0.322687029838562, -0.2699332535266876, -0.09930506348609924, 0.18209204077720642, 0.887733519077301, 1.0633282661437988, -0.7539107203483582, -1.0055272579193115, -0.4758121967315674, -1.3727192878723145, 0.5503652095794678, 0.9768749475479126, -0.8349646329879761, -0.4729483425617218, 2.612978219985962, 0.30680525302886963, -0.4971698820590973, -0.7752094268798828, 0.7029488682746887, 1.8085548877716064, 0.2338772863149643, 0.7034720182418823, 0.13381533324718475, -0.7494481801986694, -1.7108662128448486, 0.11301351338624954, -0.901695191860199, 0.3213406801223755, -0.6238200068473816, 0.5823501944541931, -0.5323470234870911, 0.3111174702644348, 0.8651257753372192, 0.9830660820007324, 0.492588073015213, -0.26943472027778625, 0.05797550827264786, 0.0683678388595581, 0.006284733768552542, -0.7973535060882568, 0.20253242552280426, -1.0061113834381104, -1.128639817237854, -0.3602786362171173, -1.5461117029190063, -0.8718245625495911, 2.7942423820495605, 0.5181755423545837, 0.4705814719200134, 0.46691203117370605, 0.7113851308822632, -1.229928731918335, -0.6437487006187439, 1.3173487186431885, -0.2616162896156311, 2.003291606903076, 0.39631137251853943, 0.9582318663597107, 0.4962838292121887, 1.201545000076294, -0.6473981142044067, -0.5774564743041992, -0.22537043690681458, 0.7956503629684448, -0.7322295308113098, -0.34919869899749756, -0.05217602849006653, 1.1545782089233398, -2.276101589202881, -0.13122332096099854, -0.0034940901678055525, 0.7198931574821472, 0.8583489656448364, 2.4641623497009277, -0.42919105291366577, -0.0699806734919548, -0.006874337326735258, 0.15676315128803253, -1.7546254396438599, -0.05051339045166969, -1.2858232259750366, -0.1788361370563507, -1.630370020866394, -1.9627692699432373, -1.2463942766189575, 1.1858021020889282, 0.6175103187561035], [0.5005456805229187, 0.2994251251220703, 1.084900975227356, 0.06655522435903549, -0.06277864426374435, -0.5227387547492981, 0.01537409145385027, 1.0663632154464722, 0.9641962647438049, 0.0674823522567749, -1.2971733808517456, -0.029529327526688576, 1.64556086063385, 0.1909942924976349, 1.5043591260910034, -1.3209595680236816, -0.14511819183826447, -0.26877132058143616, -0.09328631311655045, -1.8767369985580444, -0.20318885147571564, 1.485644817352295, 0.38528650999069214, -1.0872905254364014, -0.334878534078598, 0.795488715171814, 0.6302312612533569, 0.3748718500137329, 0.27638283371925354, -0.7050007581710815, -0.32471153140068054, 0.8859951496124268, 0.28182747960090637, -0.08361883461475372, -0.7908108234405518, 1.1521292924880981, 0.047841109335422516, 0.7916946411132812, -0.23419605195522308, -1.1433687210083008, 1.8739392757415771, 0.05233600363135338, -1.1404037475585938, -0.05481589213013649, 0.7533888816833496, -1.8324400186538696, -2.247317314147949, -0.5203990340232849, 0.6913854479789734, -0.3418886661529541, -0.15745027363300323, 0.07354051619768143, 0.08190706372261047, -1.1967904567718506, -0.34405550360679626, 0.173810675740242, 1.831252932548523, -1.3613636493682861, -1.2618783712387085, -0.037935443222522736, 1.799391269683838, 0.7293365001678467, 1.6452821493148804, 0.15475352108478546, -1.3032608032226562, -0.08969101309776306, 0.5927349328994751, 1.2029601335525513, 0.5270178914070129, 0.8727645874023438, 0.7394521236419678, -0.011697669513523579, 1.6218984127044678, 0.8901119232177734, 0.976426899433136, -1.0378737449645996, 0.9092933535575867, 0.09252388775348663, 0.9423288106918335, -0.3461318016052246, 0.7727918028831482, 0.24954582750797272, 0.47307538986206055, 0.6140487790107727, 2.490774393081665, 1.4239121675491333, -0.36205700039863586, 1.660085678100586, -2.053915500640869, 0.017761874943971634, 0.018726851791143417, 0.0436210110783577, -0.05708752200007439, -0.011002615094184875, 1.6897053718566895, 0.3647138476371765, 0.8657283186912537, 0.5496982932090759, 0.20613040030002594, -2.125006914138794, 0.5189468860626221, 2.2047035694122314, 0.05279180407524109, -0.5897909998893738, 0.7998209595680237, -0.418443888425827, -0.3024771213531494, -0.889897346496582, -0.1445009708404541, 1.0738507509231567, 0.25885245203971863, 0.8105308413505554, -0.790039598941803, -1.095589280128479, 1.8536731004714966, -0.2876759469509125, 0.5471369028091431, 1.2977559566497803, -0.10700860619544983, -0.9923677444458008, 0.15095680952072144, 0.8945554494857788, 1.0632531642913818, 0.18539844453334808, 0.5030174851417542, 1.9796831607818604, -1.3600313663482666, -1.5038871765136719], [-0.7424718737602234, -1.2189207077026367, 1.0102310180664062, -0.021703006699681282, 0.5583935379981995, -0.6871292591094971, 0.6551117897033691, 1.2664638757705688, 0.6819502115249634, 1.658722162246704, 0.9391435980796814, 0.3477816581726074, -1.832047939300537, -0.1371670961380005, -2.503960132598877, 0.571535050868988, 0.8937636613845825, -0.29219093918800354, 2.0297985076904297, -1.4180692434310913, -1.8302979469299316, 1.4005153179168701, -1.6392585039138794, -1.7082269191741943, 0.3429115414619446, -0.8547420501708984, -0.9672390818595886, 1.1235976219177246, -0.5312827825546265, 0.6837803721427917, 1.050561785697937, 0.6727798581123352, 0.5639749765396118, 1.3273015022277832, -0.3116634488105774, 0.01676183007657528, 0.17671339213848114, -0.12771080434322357, 0.624853789806366, 1.5980340242385864, -2.9518167972564697, 0.2142687290906906, 0.9826899766921997, 1.7771453857421875, 0.14351670444011688, -0.07899907231330872, -0.5087264776229858, -1.7539931535720825, -0.7806715965270996, -1.1355268955230713, 0.6617454886436462, -1.0682929754257202, -1.7984546422958374, -0.07686896622180939, -0.1339195966720581, -1.9680370092391968, -0.8330596685409546, -0.5951038599014282, 1.431915521621704, 0.4157077670097351, 0.45962584018707275, 0.7508911490440369, -0.17607976496219635, 0.45782917737960815, -0.8185219168663025, -0.8270999193191528, -1.4841828346252441, 1.2375563383102417, 0.5035213232040405, 0.9607270956039429, 0.08641303330659866, -0.34405478835105896, -1.4983396530151367, 2.134481430053711, 0.7022513747215271, 1.003947377204895, -0.033367402851581573, 0.639444887638092, -0.1057409942150116, 1.0230541229248047, 0.6302642226219177, 0.3208603262901306, -0.33513468503952026, 0.6697139739990234, -2.9017248153686523, 0.6049721240997314, -1.3026955127716064, -0.48508748412132263, 1.833580493927002, -0.3474310040473938, -0.8003436326980591, -0.3755069673061371, -1.2515827417373657, 2.0390536785125732, -0.11660704761743546, 0.6990674138069153, -2.2903268337249756, 0.8966825604438782, -1.347080945968628, -1.0348833799362183, 0.9621413350105286, -0.7324299812316895, 1.9269380569458008, -1.0928428173065186, -0.585071861743927, -0.5120554566383362, -0.3690527379512787, -0.19126203656196594, 0.7789895534515381, -1.7956904172897339, 1.410448431968689, 1.3217781782150269, -1.1772786378860474, 0.9196853041648865, -0.9607906937599182, -0.7600651383399963, -0.1852118968963623, -0.8624851703643799, -0.7076956033706665, 1.0662282705307007, -0.06040225923061371, 2.109149932861328, 1.8719432353973389, 1.881333351135254, 1.464568853378296, -0.8294231295585632, -0.3321828842163086, 0.1326683610677719], [1.3945369720458984, -2.1887435913085938, 0.05234510079026222, 1.5062892436981201, -1.2162890434265137, 0.33568525314331055, -0.3250705599784851, 0.24966338276863098, -1.0712693929672241, 0.2848719358444214, 1.1571764945983887, 0.276867538690567, 1.4668744802474976, -0.5597755312919617, -1.890359878540039, 0.11312025040388107, 1.8376781940460205, -1.4589614868164062, 0.914429247379303, -0.45627620816230774, -0.5566643476486206, -0.683754026889801, 0.23749971389770508, -1.7455142736434937, -0.33903178572654724, 0.07636924833059311, -1.8719202280044556, -1.2821491956710815, 0.927047848701477, -1.2736374139785767, -0.0492459312081337, -1.872238278388977, -1.5982950925827026, -0.033062104135751724, 0.17357264459133148, 0.8401107788085938, 0.315463125705719, 1.772408366203308, -1.3495168685913086, 1.2415978908538818, 1.0355292558670044, 0.041669026017189026, 0.535368025302887, 0.3211529850959778, 0.5671623349189758, 0.17217373847961426, 0.2985934317111969, -0.027692489326000214, -1.4879040718078613, 0.9135210514068604, -0.16283127665519714, 0.8629482984542847, 0.033769216388463974, 0.06051235273480415, 0.27840158343315125, 0.43124833703041077, 0.5104841589927673, -0.14416734874248505, 0.30892476439476013, -0.04348626360297203, -0.1459350883960724, -0.4464893937110901, 0.0272369347512722, 1.4457581043243408, -0.5730645656585693, -1.0027432441711426, 0.7895936369895935, 0.5767456889152527, -1.9835715293884277, 0.5004451274871826, 0.8632112741470337, 1.3323253393173218, -0.737375795841217, 1.157801866531372, -0.8212834000587463, -0.19563235342502594, -1.901475191116333, -0.4217582941055298, 0.9816564321517944, -1.479397177696228, 1.1614608764648438, -0.05338146165013313, -0.11433286219835281, 2.8312227725982666, 1.2597672939300537, 2.0587470531463623, 0.0019865341018885374, -0.440550833940506, 0.2688768804073334, 1.4584226608276367, -0.349351167678833, 0.3131820261478424, 0.6567233800888062, -1.3477224111557007, -0.023386653512716293, -0.10239271819591522, 0.784195065498352, -0.10257235169410706, -0.4073459506034851, 1.643672227859497, 0.2002735435962677, 0.20711347460746765, 0.23344570398330688, 0.2877582013607025, -1.0257357358932495, 0.41943392157554626, -0.49459710717201233, -0.7137543559074402, -2.7868804931640625, -1.697707176208496, -0.9444709420204163, -0.04469746723771095, 0.20901815593242645, 0.8990409970283508, 0.37585651874542236, 0.28758710622787476, -1.805021047592163, 0.5164909958839417, -0.8079023361206055, 1.9197683334350586, 0.6280403137207031, -1.2155206203460693, -0.5834160447120667, -0.5227894186973572, 1.314574956893921, 0.16879040002822876, -1.7786537408828735, 0.18329556286334991], [-0.4966244399547577, 1.2313816547393799, -0.2668014466762543, 0.6048604846000671, 0.06277398020029068, 0.42733269929885864, -0.08788275718688965, 0.49810945987701416, 1.3166922330856323, 0.061130572110414505, -0.31752198934555054, 1.8993703126907349, -2.0343661308288574, 1.4929218292236328, 1.0334086418151855, 0.4534750282764435, -0.7577546834945679, -0.9207958579063416, 0.54932701587677, -0.6315317749977112, 0.689294159412384, -0.06649930030107498, -0.3867926299571991, 1.0053684711456299, 0.7887251377105713, 0.1172695979475975, -1.021459698677063, 0.9041957259178162, -1.3456425666809082, 0.22166398167610168, 0.12300646305084229, 0.5390583276748657, 0.8014519810676575, -2.123492479324341, -0.11468736827373505, 0.15153098106384277, 0.7875960469245911, -1.0029059648513794, -0.1042182520031929, 1.0467031002044678, -2.4547653198242188, -0.6538052558898926, -0.2987419366836548, 1.0239503383636475, 1.1910592317581177, -1.5908957719802856, -1.6066313982009888, -1.5466076135635376, 1.559889316558838, -0.8933194279670715, -0.29235294461250305, 1.9552011489868164, 0.7001263499259949, 0.7509555816650391, 1.0982153415679932, 1.2376203536987305, -0.4220498502254486, -0.5548956394195557, -1.324030876159668, -0.43655431270599365, -0.6807094812393188, 0.5006273984909058, -0.05953628569841385, -1.5166027545928955, 0.205842524766922, -0.17399947345256805, -0.5955467820167542, 1.5185104608535767, -1.4000071287155151, -0.23812861740589142, -0.13493512570858002, 0.2105790376663208, -1.347205638885498, 1.3172003030776978, 0.35163912177085876, -1.3043832778930664, -0.8459954261779785, 0.16731494665145874, 0.5733267068862915, 0.39658722281455994, 0.19683107733726501, 0.7305116057395935, -0.6485216617584229, 1.2457787990570068, 1.1375524997711182, 1.0702468156814575, 0.17949038743972778, 0.4980950355529785, -0.17337098717689514, 0.10187989473342896, 0.6197662949562073, -0.4018521010875702, 0.45491722226142883, -0.36596357822418213, 0.42282986640930176, 1.2103244066238403, 1.176345705986023, 1.6268103122711182, 0.32917264103889465, 0.5071425437927246, 2.506303548812866, -0.2875687777996063, 0.526745080947876, -0.7376765012741089, 0.25610002875328064, 0.48473188281059265, 0.6260219216346741, 0.45264729857444763, 1.531980037689209, -1.9734547138214111, 0.7866997122764587, 0.7004535794258118, -0.8448794484138489, 0.03626934811472893, 0.6085920929908752, -0.893048107624054, -1.2910512685775757, 0.913415789604187, -0.26455461978912354, 0.20452111959457397, 0.4485395550727844, -0.33735185861587524, 0.37849998474121094, 0.6644996404647827, -0.5090625882148743, 0.3277113139629364, 0.9109209179878235, -0.6509144306182861], [-0.8622997999191284, -0.28894248604774475, 0.4359614849090576, 0.021205240860581398, 0.12671354413032532, 1.5253835916519165, -0.13679493963718414, -1.1319619417190552, -1.2587039470672607, -1.5489060878753662, 0.9239091873168945, 0.9173169136047363, -0.4679183065891266, 0.9546884298324585, -0.16825313866138458, -0.8558207154273987, 0.3187078833580017, 2.579331159591675, -1.3293901681900024, 1.9373157024383545, -0.3650056719779968, 0.3275472819805145, -0.8506484031677246, 0.6439589262008667, 0.6084907650947571, 1.3652193546295166, -1.9098089933395386, 0.34830528497695923, -0.7113310098648071, 0.35139378905296326, -0.8710262775421143, 0.1350686401128769, 0.3435118496417999, 0.4394950270652771, 1.4697099924087524, 0.6106778979301453, 1.1114505529403687, -1.2655742168426514, -0.6855363249778748, 0.1592155247926712, -1.6626899242401123, -0.7620053887367249, -1.0520919561386108, -1.8537813425064087, -1.1964325904846191, -0.029976943507790565, -0.8385424613952637, 1.7752307653427124, 0.8027554154396057, -0.3372238576412201, -0.012204539962112904, -0.25467124581336975, -0.889041006565094, -1.3287030458450317, -1.0764377117156982, 0.5906882882118225, 1.6937216520309448, 0.23586268723011017, 0.6884283423423767, 0.45325517654418945, 1.4733481407165527, -0.034521374851465225, -1.4619836807250977, -1.268178105354309, -0.633189857006073, -1.431470513343811, 0.839670717716217, 0.5528461933135986, -1.373809814453125, 0.7205171585083008, 0.8063639402389526, 0.11253715306520462, -0.45798560976982117, -1.307938575744629, -0.8779356479644775, -0.2497999370098114, 0.9346945285797119, 0.4788982570171356, -0.3617997467517853, -0.11315979808568954, 0.4724235236644745, 0.0004920032224617898, -0.10234405845403671, 0.9032514095306396, 1.2068705558776855, 0.8127568364143372, 0.7952205538749695, -1.4628241062164307, 0.6922202706336975, 1.3046839237213135, -0.6363561153411865, -2.178997278213501, -0.8022419810295105, 0.771980345249176, 0.6655300259590149, 0.2197171151638031, -1.013854742050171, -0.8101763725280762, 0.5296400785446167, 2.3126699924468994, -0.6280056834220886, 0.9624603390693665, 1.2841062545776367, 0.8174829483032227, -1.4888834953308105, 0.6589670181274414, -0.012415238656103611, -0.588050365447998, 0.45201611518859863, -0.09706784784793854, -1.5628182888031006, -1.825595736503601, 1.8139348030090332, 1.115209937095642, -0.8297451138496399, 0.4735676646232605, -0.6620298624038696, -1.6222875118255615, 0.5243555307388306, 0.24453701078891754, -1.5546504259109497, -0.7256485223770142, 0.09430896490812302, -0.6529589891433716, 0.18041610717773438, 0.3832359313964844, -1.1890337467193604, 0.7683270573616028], [0.7565594911575317, -0.2842142879962921, -0.8153767585754395, 0.7062024474143982, -0.1407870352268219, -1.6211754083633423, 0.04181094840168953, -0.024200741201639175, -0.5699110627174377, 0.8400667905807495, 0.8568781018257141, 1.0137537717819214, 0.10657360404729843, -1.2597177028656006, 0.6957716345787048, 0.8723849058151245, 0.43731310963630676, -0.9235053658485413, -0.20749621093273163, 0.5771776437759399, -0.6272879242897034, 0.1573866307735443, -0.9691017270088196, 0.5228204131126404, 0.33929136395454407, 0.1055503860116005, -0.46769481897354126, 1.0021698474884033, 0.4227342903614044, 2.7646422386169434, 2.2100183963775635, 0.3883391320705414, 0.11304788291454315, -0.4687776267528534, -0.09322092682123184, 0.16977404057979584, 0.13847576081752777, -0.945061981678009, 0.30396321415901184, 1.9629589319229126, 0.8323060870170593, -0.4724487364292145, 0.4890528917312622, -0.3143205940723419, 1.6471184492111206, -0.41723647713661194, 1.1079589128494263, -1.5556787252426147, -0.1865662932395935, -1.267609715461731, -0.3961016535758972, -1.5703731775283813, 0.7472653985023499, -0.27126601338386536, -1.03293776512146, -1.027147889137268, 0.3748215138912201, 1.198794960975647, -0.3508644998073578, 1.6967021226882935, 0.09976844489574432, -0.7786556482315063, 0.3228370249271393, 0.6728213429450989, -0.2035776674747467, -0.9995637536048889, -0.501484751701355, -0.11330914497375488, -1.7765358686447144, -1.6152622699737549, 0.6334791779518127, 0.8015154004096985, 2.2484612464904785, 1.653214454650879, 0.774218738079071, -0.3769580125808716, 0.35327059030532837, -0.8906023502349854, 0.5397441387176514, 0.30950483679771423, 0.6646400094032288, 1.1101816892623901, -0.3142932951450348, 0.21637092530727386, -0.1475284993648529, 1.077648639678955, -0.5567023754119873, -0.2105904072523117, 0.38861870765686035, 0.04666053131222725, 1.7077122926712036, 0.07224210351705551, -0.826228678226471, 0.5678837895393372, -0.3435814380645752, 0.567779004573822, 0.1821478307247162, -0.15004447102546692, 0.9502541422843933, 0.8161904215812683, 0.15351226925849915, -1.7952932119369507, -0.2212299406528473, 0.14886675775051117, -1.231119990348816, -0.9274924397468567, -0.2820729613304138, -1.7886183261871338, 0.011465666815638542, 0.23748326301574707, -0.2678654193878174, 0.6040937304496765, -0.3988804817199707, 0.5910070538520813, 1.41261625289917, 0.2904631793498993, 1.019715666770935, -0.42408955097198486, 0.7622535228729248, -0.22174187004566193, -1.5977606773376465, 0.5514046549797058, 0.23446890711784363, -1.1904088258743286, -0.6122871041297913, 0.27604520320892334, -1.3711291551589966, 0.622901976108551], [0.8234130144119263, -0.45438286662101746, 0.029662195593118668, 0.33278435468673706, 1.2291139364242554, -0.37249690294265747, -1.1425329446792603, 1.1485577821731567, -0.08559730648994446, 0.7527541518211365, -0.12157316505908966, 0.20399636030197144, -0.701350212097168, -0.11219246685504913, -0.2139553725719452, 0.22188915312290192, 1.0217722654342651, -0.6118366718292236, 0.9258009791374207, 1.7263582944869995, -1.4250184297561646, -0.20353932678699493, 0.09506019949913025, -1.2100439071655273, 0.4768548309803009, -0.28727948665618896, 1.8286372423171997, -0.06963665783405304, -1.3357231616973877, 0.34710511565208435, -0.8795841932296753, 0.8727853298187256, -0.9131662249565125, -0.9375152587890625, -1.1034985780715942, -1.7978708744049072, 0.4171693027019501, 0.45526042580604553, -0.6800351738929749, 0.08850879967212677, 0.5279372930526733, 0.5875232815742493, -0.3606540858745575, 1.4748457670211792, -0.5989969968795776, -0.54029780626297, 2.917659044265747, 0.952104926109314, -1.2472792863845825, -0.525526762008667, 1.0520548820495605, 0.4289451539516449, -0.2916528582572937, -1.805116891860962, 1.6124951839447021, -3.24283504486084, 0.021313214674592018, 0.08529645949602127, -1.98397696018219, -0.28731393814086914, 0.16497382521629333, 0.2187376618385315, 1.84028959274292, 0.2906523048877716, -0.04874328151345253, 0.3980984389781952, -0.559052050113678, -1.1424223184585571, 1.2992092370986938, -1.7535028457641602, -1.3840034008026123, 0.2357650101184845, 0.754631757736206, 1.5123305320739746, 0.5486130118370056, 1.385111927986145, -0.24362847208976746, -1.6014431715011597, 0.30476754903793335, 0.41046881675720215, 0.7277231216430664, 0.062051594257354736, -1.612120509147644, -0.40359851717948914, -0.9891401529312134, -0.0911535993218422, 0.9421119093894958, -1.074264645576477, 0.11011788994073868, 0.9335228800773621, 0.27386337518692017, -0.6655332446098328, -1.8918635845184326, 0.8801284432411194, 0.9216731190681458, -1.544969081878662, -0.3422028720378876, 0.1755581796169281, 0.08533183485269547, -0.13274843990802765, -0.45594868063926697, 0.478941947221756, -2.192838191986084, -0.31386798620224, -1.1803150177001953, -0.42789748311042786, 1.0601603984832764, -1.0027046203613281, -0.5249610543251038, -0.09650009870529175, -2.4666409492492676, 0.4840660095214844, -1.3849644660949707, 0.04067175090312958, 0.3862036466598511, 2.5837252140045166, 0.6001251339912415, -0.6819326281547546, 1.6772416830062866, 1.5989100933074951, 0.020870793610811234, -0.6309933066368103, -0.2602255940437317, -0.3790099620819092, 1.4187376499176025, -0.6622565984725952, 2.2648847103118896, 0.4031396508216858], [-1.1945874691009521, 0.4558647871017456, 0.9383184313774109, 1.212079405784607, -0.28974902629852295, 1.5669009685516357, 0.2285921573638916, -1.4099980592727661, -0.2693069577217102, 0.10400794446468353, -0.07222166657447815, 1.4977749586105347, -0.5880477428436279, -0.15689627826213837, -0.667439341545105, 2.0244245529174805, -0.20697849988937378, 0.17300641536712646, 0.7556427717208862, 0.6582723259925842, 0.9434027075767517, -0.2649911046028137, 0.605088472366333, 0.1439952552318573, -0.5105141997337341, -0.28130072355270386, 1.1042755842208862, -0.3748871982097626, -0.23001284897327423, 0.7186799645423889, -1.057750940322876, -0.497955858707428, -0.43436598777770996, 0.010255459696054459, 0.36472851037979126, -1.4192944765090942, 0.14186538755893707, -1.0029546022415161, 1.0493091344833374, 1.3483484983444214, -0.2822191119194031, 1.3099333047866821, 0.4920402765274048, 0.5509594678878784, 0.32638323307037354, 0.5303411483764648, 0.1815379112958908, -1.0262908935546875, -0.3961615264415741, -1.733276128768921, 0.904141366481781, -0.6834386587142944, 2.2628719806671143, -0.12155424058437347, 1.4770642518997192, -1.5542072057724, -0.4994603395462036, -1.4385368824005127, 2.1182425022125244, -0.2815050184726715, -1.480025291442871, -0.34606674313545227, -0.5678040385246277, 0.7932085990905762, -0.6945303678512573, 0.5079519152641296, 1.0070222616195679, -1.4751129150390625, -0.3134469985961914, -0.2527419626712799, -2.0143141746520996, -1.376089096069336, -1.974181890487671, -1.6810569763183594, 0.6167836785316467, 0.6950915455818176, 0.5009627342224121, -1.07069730758667, 0.8685047626495361, 1.5380926132202148, 1.268444538116455, -0.03067629784345627, 0.7004007697105408, -0.47838112711906433, -1.2214518785476685, 0.21222688257694244, 0.530742883682251, 1.2348439693450928, 0.18234938383102417, -0.6468806862831116, 0.8036013841629028, 0.3978578448295593, -0.5977055430412292, 0.20074769854545593, -0.7522698640823364, 0.4217672049999237, 0.28490114212036133, 0.2009502798318863, -0.2971593141555786, -0.14243347942829132, 1.063400149345398, 0.7209349870681763, -1.6237396001815796, -1.5509576797485352, -0.356570303440094, 1.2846921682357788, -0.9160302877426147, 1.6883113384246826, 1.772749423980713, -0.6681764125823975, -0.04766669124364853, 1.028183937072754, -0.23457689583301544, 0.467874675989151, 0.9137417674064636, -1.7421362400054932, 0.5139992833137512, -0.7239978909492493, -1.448677897453308, 1.687333345413208, -0.6004539132118225, -0.6483697295188904, 2.6918888092041016, 0.30367034673690796, 0.648576021194458, -0.8575601577758789, -0.8455260992050171, -0.21002189815044403], [-1.5857831239700317, 0.6191878318786621, -1.28817880153656, -0.5002008676528931, -0.22529470920562744, -1.3406826257705688, 0.8523648977279663, -0.8433981537818909, 0.9303467869758606, -1.2028948068618774, 0.709297239780426, 0.19952544569969177, 1.2871406078338623, -0.18502403795719147, 0.5597273707389832, -0.8075486421585083, 0.9509680271148682, 1.722025752067566, 1.8724689483642578, 1.9858533143997192, -1.0973303318023682, -0.6884850263595581, -0.3010598123073578, 0.22008781135082245, -0.7930852770805359, 0.16909897327423096, -0.7880614995956421, -0.7760675549507141, 0.0869261845946312, -0.07564068585634232, -0.5835213661193848, -0.5971148014068604, -0.0031460868194699287, 0.5957187414169312, 2.0583930015563965, 0.4656514823436737, 2.24589467048645, 1.598362684249878, 0.4466097950935364, -1.205599308013916, -0.04482110217213631, -0.017014699056744576, 1.1050009727478027, 0.7091229557991028, -0.6781530976295471, 1.1391713619232178, -0.972657322883606, 1.4188320636749268, 1.9082043170928955, -0.786516010761261, -1.3284480571746826, 0.2235911786556244, -0.0757441595196724, 0.0810021236538887, -0.850917637348175, -0.1915479451417923, 1.5671764612197876, 0.2619302570819855, -0.36642399430274963, 0.1577862650156021, -1.621389389038086, 0.17841677367687225, 1.7388654947280884, 0.05399373173713684, 0.34080901741981506, -0.20424893498420715, 0.5624572038650513, 1.5335655212402344, 0.37377217411994934, 2.0156517028808594, 0.5408284664154053, -0.6263664364814758, -0.41381651163101196, -1.043474555015564, 0.14283086359500885, 1.1305166482925415, -0.05838265269994736, 0.9340142607688904, 1.0027779340744019, 1.4287166595458984, -0.552708625793457, 1.1125274896621704, 0.5718362331390381, -0.5459086298942566, 1.603287696838379, 0.7304250001907349, -0.8506794571876526, 0.4993237257003784, 0.061021946370601654, 2.1839489936828613, -0.670260488986969, -1.6799062490463257, 2.1215200424194336, 1.055606484413147, 0.39665499329566956, -0.743718147277832, 0.15854361653327942, -0.7515149116516113, -0.6631274819374084, -0.16467516124248505, -0.020262449979782104, 2.2592923641204834, 1.429681420326233, 0.5516888499259949, 0.41037294268608093, 0.6744397878646851, -0.031419891864061356, -1.8019579648971558, 1.3296648263931274, 1.2999908924102783, 0.002032372634857893, 0.7405509948730469, 0.6970211267471313, 0.7566086649894714, -1.468945860862732, -1.3143250942230225, -1.9884207248687744, 0.33486250042915344, -0.34446045756340027, 0.15139363706111908, -0.5952470302581787, -0.017044642940163612, 2.0817043781280518, -0.20747621357440948, -1.130898118019104, -1.3242956399917603, 0.47397738695144653, -0.30208003520965576], [-1.5755468606948853, -0.9428077340126038, 1.1392385959625244, 0.06089157983660698, 0.17002959549427032, -2.123746156692505, -0.3676760196685791, -0.03207795321941376, -0.9083617329597473, -1.2127840518951416, -0.28831562399864197, 1.1838897466659546, -0.2435285896062851, -1.8186345100402832, 0.209671750664711, 0.8332332372665405, -1.41754150390625, 0.5321266651153564, -0.010736863128840923, 0.5806208848953247, -0.5555709004402161, -0.11440341919660568, -0.19031567871570587, 0.2466910183429718, 0.08195528388023376, -0.3207704722881317, -0.8352629542350769, -1.8260830640792847, 3.093604326248169, 0.27998143434524536, -0.003372374689206481, -1.1731356382369995, 0.0009064293699339032, 0.6008813381195068, 0.5452737212181091, -0.9109047651290894, -0.671276867389679, 0.0682809129357338, 0.7137423157691956, 0.46047982573509216, -1.456971526145935, 0.6499326229095459, 0.834935188293457, 0.3100498616695404, 0.5388136506080627, 0.9835342168807983, -0.27651843428611755, -0.116878941655159, 0.2267395257949829, 0.8592209815979004, 1.0510543584823608, 0.5698727369308472, 0.27379027009010315, -0.5751916766166687, -1.077896237373352, 0.3508351743221283, -1.8587828874588013, -1.1705715656280518, 0.2217796891927719, -1.3194692134857178, -0.9248193502426147, 2.9294896125793457, 0.381817489862442, -1.087397575378418, -0.920177161693573, -0.2655741274356842, -0.5813751220703125, 0.18247927725315094, 1.2600780725479126, -1.2352631092071533, -0.062300216406583786, 0.4016275107860565, 0.6508580446243286, -1.564854621887207, -0.7898463606834412, -0.43431153893470764, -0.08660385012626648, 0.6208560466766357, 0.9874556660652161, -0.38403674960136414, -1.1883455514907837, -0.9698981642723083, 0.4157487154006958, 0.08557858318090439, -0.261001318693161, 1.558609962463379, -1.582410454750061, 0.6806780695915222, 1.357163906097412, -1.3596247434616089, 0.36845797300338745, 0.3665068447589874, -0.5396491885185242, -0.05651214346289635, -1.4463623762130737, -0.3786013126373291, 1.9221885204315186, 2.1952574253082275, 1.016908884048462, 0.6602616310119629, 0.80146324634552, 1.400316834449768, 0.6344254612922668, -0.5199176669120789, -1.1751617193222046, -1.2611184120178223, -0.13378803431987762, 0.37588921189308167, -0.9352788329124451, 1.5756686925888062, -0.05593233183026314, 0.15485554933547974, 1.8676421642303467, -1.250511646270752, -0.8290877938270569, -1.44438898563385, 2.0229415893554688, 0.07554257661104202, -0.20148210227489471, 0.9983507990837097, -2.211559295654297, -0.0073060509748756886, 0.7676624655723572, 2.000816822052002, -0.963685929775238, -1.4856377840042114, -0.1695987731218338, -0.25604447722435], [1.5785367488861084, 1.639958381652832, -0.6131051182746887, 1.8151607513427734, 1.203959584236145, -1.0463110208511353, 0.13052591681480408, 0.9949732422828674, 1.8185137510299683, 1.3420051336288452, 0.359113872051239, 1.1965166330337524, 0.27006399631500244, 1.0621556043624878, -0.2142377346754074, -0.929552435874939, 0.7542343139648438, -1.0455601215362549, -0.5149068832397461, -0.3888445198535919, -1.098559856414795, 1.1077852249145508, 0.2457544207572937, 1.3463608026504517, -0.8406965136528015, 0.5301397442817688, -0.34608352184295654, -0.8420020937919617, -0.7230731248855591, 0.46959567070007324, -0.7733767628669739, 2.480560541152954, 1.6753687858581543, 0.7132502794265747, 0.4058551490306854, 0.8935560584068298, -1.4380528926849365, 0.09622196108102798, 0.04001499339938164, -0.1951059252023697, 0.037045758217573166, -0.38226914405822754, 0.3769805431365967, -0.7931156754493713, 0.7442998886108398, 2.0409812927246094, 1.3738960027694702, 0.22556477785110474, -1.6227854490280151, 0.12932904064655304, 1.2243026494979858, -0.6322495341300964, 0.03548334911465645, 1.288041114807129, -1.3347562551498413, 0.8168092966079712, -1.9795677661895752, -0.4199254810810089, -0.0204001534730196, -0.6601029634475708, 0.1843864619731903, -0.3885456621646881, 1.4519286155700684, 0.15240924060344696, 0.6690459251403809, -1.0425231456756592, 0.259880393743515, -2.5560083389282227, -0.913844645023346, 0.8040894269943237, -0.6182993650436401, 0.0523531399667263, -0.1302090734243393, -1.0303747653961182, -2.01867413520813, -0.4286888837814331, -0.4641100764274597, -1.1284234523773193, -1.431792974472046, 0.26959022879600525, 1.3489980697631836, -2.9283764362335205, -0.1837066262960434, -1.9258490800857544, 0.1746651977300644, -0.8306698203086853, 0.984034538269043, -1.3289003372192383, -1.245200753211975, 0.4988853633403778, -0.8611681461334229, 2.6476216316223145, 0.8270875215530396, 1.1467626094818115, 1.7751399278640747, -0.5789220333099365, -0.6015247702598572, -0.05920518562197685, -0.023683827370405197, 1.1278154850006104, -0.3779609203338623, 1.1128720045089722, 0.3147582411766052, -0.6066813468933105, -0.10788961499929428, 0.18286874890327454, 1.4679334163665771, 0.22093422710895538, -0.5834985971450806, 1.0739688873291016, 0.5035768151283264, 1.6121790409088135, 0.08248542994260788, 0.27828314900398254, -0.004346068948507309, -0.22005733847618103, -0.0037791042122989893, -1.1655858755111694, -1.1619020700454712, -0.4974851906299591, -1.009321928024292, -0.16625040769577026, -1.1723287105560303, 0.45229846239089966, -1.0291423797607422, 1.1942458152770996, 0.5191335082054138, -0.40493106842041016], [-1.3377114534378052, 0.25706878304481506, 1.3485842943191528, 0.3754171133041382, 1.4543068408966064, -0.3423083424568176, 0.7952388525009155, -1.8481943607330322, 0.48833566904067993, -0.960098922252655, 0.7166962027549744, -0.7986050844192505, 0.9542317390441895, -1.0401989221572876, 0.4592195153236389, 1.320810317993164, -1.3474177122116089, -1.592621088027954, 0.25923487544059753, 0.07554244250059128, -0.10406024008989334, 0.5059288144111633, -0.46413475275039673, -0.5485216975212097, -2.030606985092163, 0.21031570434570312, -1.2152246236801147, -0.8277778029441833, 0.7310110330581665, -2.0542242527008057, 0.13864341378211975, -0.7695790529251099, 1.631450891494751, 1.1642166376113892, -0.5246835350990295, -0.17324237525463104, -1.1051894426345825, 1.1712599992752075, -0.057612061500549316, -1.1542571783065796, -0.14885610342025757, 1.0070146322250366, 0.6746516227722168, 0.3736543357372284, -0.1276867836713791, 0.5448930263519287, 0.7161319255828857, 0.5587740540504456, 0.05145236849784851, 0.4685536324977875, 1.1867631673812866, 0.3021952509880066, 0.13267222046852112, 0.33141499757766724, -1.317785620689392, -0.35537460446357727, 0.453124076128006, 0.4073197543621063, -0.4345951974391937, 0.0005192619864828885, 0.7165915369987488, 0.327172189950943, 2.530574321746826, -0.2690153121948242, 1.0917176008224487, 0.18448974192142487, 0.6128479838371277, -1.5938208103179932, 0.4894081652164459, 1.6278809309005737, 0.3545002043247223, -1.568317174911499, -1.2159359455108643, -0.2954398989677429, 1.3038190603256226, -0.427724689245224, -0.6524084210395813, -1.0739104747772217, -0.44400519132614136, 0.88075852394104, -0.5661770105361938, 1.4789319038391113, 0.46909624338150024, -0.9881116151809692, 0.8222607970237732, -1.2633403539657593, -1.8499975204467773, 0.7026697993278503, 0.3344527781009674, 0.048074670135974884, -0.24354541301727295, 2.020265579223633, 0.23259004950523376, 2.5409204959869385, 0.13614225387573242, 1.591578722000122, -0.0424920916557312, -0.17536096274852753, -0.13970515131950378, 1.9141076803207397, 0.55314701795578, 0.9042267203330994, 0.599003791809082, -0.5166060328483582, 2.22247576713562, -0.6998989582061768, -0.08723339438438416, 0.6638195514678955, 0.03266027197241783, 0.32393360137939453, -0.19419153034687042, 0.2880812883377075, -0.4963734745979309, 0.09578946232795715, 0.28572043776512146, -0.014994809404015541, -0.0407218299806118, -0.4250965118408203, 0.9596251249313354, 2.545658588409424, -1.8803925514221191, -0.49698349833488464, 0.6945976614952087, -1.2749783992767334, -0.14454522728919983, 1.0123558044433594, -0.36770132184028625, 0.060329701751470566], [0.6887924075126648, -0.9846668839454651, 0.666768491268158, 0.31616050004959106, 0.7040860056877136, 0.8226658701896667, -1.6791208982467651, -0.8587191104888916, 1.0394731760025024, 0.20550915598869324, 1.1138076782226562, -0.6152858734130859, 1.3603044748306274, 1.2214604616165161, 0.753926157951355, -1.5142079591751099, 1.605059027671814, 0.09696948528289795, -0.29074805974960327, -0.38607552647590637, 1.1341543197631836, -0.4781244695186615, -0.030219223350286484, 0.01650373451411724, 0.16519950330257416, -2.486358642578125, 0.5549640655517578, -1.0385246276855469, -0.2924730181694031, 0.8825108408927917, -2.562941312789917, 0.7627599835395813, 0.03967634215950966, 0.3144569396972656, -0.23902744054794312, 0.01613703742623329, -0.9818000197410583, -0.9365770220756531, -0.2592877149581909, -0.24672017991542816, 0.07966995239257812, -1.7622612714767456, -0.0363052636384964, -0.16610948741436005, -0.46996936202049255, 2.1928930282592773, -1.0611869096755981, -0.1278000771999359, -0.022615687921643257, -0.22128057479858398, -0.45706889033317566, 0.4058537781238556, 0.6007781624794006, 1.3737905025482178, -0.18886317312717438, 0.9687783718109131, -0.05441758781671524, 0.24707554280757904, -0.7123536467552185, 0.6448745131492615, 0.41913896799087524, 0.4852672517299652, 0.18605667352676392, 1.1729391813278198, -0.0693809762597084, -1.6597579717636108, 1.013068437576294, 0.8790959715843201, -0.10346178710460663, -0.11946871131658554, -0.5214446783065796, 0.7128600478172302, 0.38445305824279785, -2.5094571113586426, -0.11176467686891556, 0.5761526226997375, -0.9025280475616455, -0.13201121985912323, 0.2123386263847351, -2.1062684059143066, 0.4561033248901367, -0.11367876082658768, 0.42730656266212463, 1.4620054960250854, -3.3337340354919434, 0.5978928804397583, -0.10702349245548248, 0.3614181578159332, -0.5913426876068115, 1.527833342552185, -1.9984411001205444, 1.1376434564590454, 0.07489565759897232, 1.01483952999115, 1.6735508441925049, 0.0770476758480072, -1.2736144065856934, 0.23472042381763458, -1.7312275171279907, 0.17180640995502472, 1.2654529809951782, -1.680354356765747, -0.8734272122383118, -0.5046138167381287, -1.5517674684524536, 0.5513091683387756, -0.3189827799797058, -0.48866304755210876, -0.0813043862581253, 1.612941026687622, -1.5961894989013672, 0.20231162011623383, -0.21451237797737122, -1.05836021900177, -1.5728137493133545, -0.014833931811153889, -0.511293888092041, -0.6254189014434814, 0.3794655203819275, -0.6952444911003113, -0.8004603981971741, 0.2579210102558136, -0.3171803057193756, 0.5095694661140442, 1.0294729471206665, 0.23436714708805084, 0.4265197515487671, 0.08777403086423874], [1.1887648105621338, 0.5410352945327759, -0.9704841375350952, 1.3201912641525269, 1.1666866540908813, 0.4476780891418457, -0.7257848381996155, 1.4934841394424438, -2.2951571941375732, 0.6691559553146362, 1.6036452054977417, -0.48699283599853516, -0.9808226227760315, 0.4680089056491852, 1.966277837753296, 0.8492833971977234, -1.1442912817001343, 0.6745782494544983, 0.28090333938598633, 1.088193655014038, 0.8573925495147705, -0.16767115890979767, -0.685820460319519, -1.5498242378234863, 0.20644868910312653, 0.4303295612335205, -0.4562973976135254, -1.1349059343338013, -1.421937346458435, 0.7411638498306274, -0.24526996910572052, -0.670896053314209, -0.09056127816438675, -0.3613799810409546, 0.5283731818199158, 0.2284502387046814, -1.6080350875854492, 0.9948897957801819, 0.5069026947021484, 0.17954300343990326, 0.19466562569141388, -1.2993550300598145, 1.4577093124389648, -1.3828779458999634, -0.2263706475496292, -0.03568267077207565, -1.1148051023483276, -1.2048468589782715, 0.3739795386791229, 1.1171048879623413, 0.7125608325004578, 0.6398584246635437, -1.5805257558822632, -1.19574773311615, -0.05394084006547928, 0.2484622448682785, -0.22864368557929993, 0.3588763177394867, 0.16956008970737457, -0.8448008894920349, 0.9996290802955627, -1.142316222190857, 0.26813703775405884, 1.9375675916671753, -0.06191238760948181, -0.9015307426452637, 0.6208108067512512, -0.5248680114746094, -1.335205316543579, -1.6210371255874634, -1.0489743947982788, 0.022995127364993095, 2.224052667617798, -0.537312924861908, -1.0247565507888794, -0.6257874369621277, -2.0738275051116943, -0.7273036241531372, -1.333809494972229, -0.023312432691454887, -0.3870801329612732, -0.5266139507293701, 0.20074044167995453, -0.2663068473339081, 0.8952712416648865, 1.4032014608383179, -1.557590365409851, -1.6178526878356934, 1.4300521612167358, -0.18698011338710785, -1.9111895561218262, -0.5206468105316162, 1.534949541091919, -1.6409441232681274, -1.9246926307678223, -1.359149694442749, 0.9230480194091797, 0.04758262634277344, -0.4590280055999756, 1.2024742364883423, -0.8498280048370361, 0.08456544578075409, 0.5511307120323181, -0.36776813864707947, 1.5746608972549438, -0.12355460971593857, -0.2983334958553314, 0.4472513496875763, 0.24374422430992126, -0.4940873682498932, -0.4164245128631592, 2.729400873184204, 0.7803682088851929, 2.262974262237549, -0.480495810508728, -1.6739418506622314, -0.5535395741462708, -0.48546820878982544, 0.11802881956100464, -0.11757536232471466, 0.7616514563560486, 1.5157207250595093, 0.06406471133232117, 0.6681580543518066, 1.6664950847625732, -0.21674352884292603, 0.6407040953636169, 1.3832281827926636], [-0.23688405752182007, 0.6801280379295349, -1.8626128435134888, -0.6871546506881714, -0.7788417935371399, 0.6733531355857849, -1.1900627613067627, 1.1765055656433105, 0.6464734673500061, -0.19419577717781067, -2.0577926635742188, 1.068152666091919, -0.2398836314678192, -0.10910467803478241, -0.7974446415901184, 0.5407590270042419, 2.378960132598877, 1.2896369695663452, -0.5156596302986145, -1.493708848953247, 0.5684201717376709, -0.40867993235588074, -0.30322620272636414, -1.2022324800491333, -1.9227136373519897, -0.3816221356391907, 0.16811910271644592, 0.20796872675418854, 0.6634092926979065, -0.6038694977760315, 1.0860378742218018, 0.9704855680465698, -0.7040146589279175, 0.2869943678379059, 1.1129708290100098, 0.6796562671661377, -0.04328425973653793, -0.6300719976425171, -0.6313930749893188, -0.8472609519958496, -0.20194387435913086, 0.2389465570449829, 0.9416621327400208, 0.6829115748405457, -1.5635795593261719, -0.25587204098701477, -0.12357993423938751, -0.8520985245704651, 1.0881773233413696, 0.4745408594608307, 1.3671268224716187, 1.743126392364502, 0.6017523407936096, -1.542762279510498, -0.4394288659095764, -0.6776553988456726, -0.9586538076400757, 0.3842915892601013, 1.2446125745773315, 0.8807055950164795, -0.9848883748054504, -1.3649977445602417, 1.4700870513916016, -0.5817782878875732, 0.44979608058929443, 0.6722607016563416, -1.2885578870773315, -0.8538572192192078, -0.4968020021915436, -0.09451235085725784, 1.5111082792282104, -0.757744312286377, -0.17446008324623108, 0.2750459611415863, -0.7270087003707886, 0.2028048187494278, 1.1634060144424438, -1.324052095413208, -0.7997023463249207, -0.5110187530517578, 0.7862169742584229, 0.13486157357692719, -0.6039005517959595, 1.5824552774429321, -0.796089768409729, 0.3791237473487854, 0.37608587741851807, 0.5973010659217834, 1.1444754600524902, -0.4200997054576874, -0.3353082239627838, -0.36876264214515686, -0.11947352439165115, -1.766525387763977, 0.8464137315750122, -1.0581374168395996, 1.2900757789611816, 1.2289224863052368, 0.40707534551620483, 0.20511892437934875, 0.012146368622779846, -0.5621189475059509, -0.380748450756073, -0.5614645481109619, 0.14501038193702698, -1.3414679765701294, -0.5154504776000977, -0.5283284187316895, -0.8057083487510681, -0.025226330384612083, -0.27982404828071594, -0.7693358659744263, 0.7548913955688477, 0.8043102025985718, 0.8778738975524902, -1.5420949459075928, -1.6868197917938232, -1.440804123878479, 0.9768102765083313, -0.996105432510376, -0.664021909236908, -0.06550175696611404, -1.0091811418533325, -0.1064128503203392, 1.3174686431884766, -0.17304390668869019, 0.9261960983276367, 1.5333449840545654], [1.7902677059173584, -0.8600616455078125, 1.072457194328308, -0.37420397996902466, -0.46289485692977905, -0.10521931946277618, -0.7948971390724182, 0.6381773352622986, -0.04903338849544525, 1.398116946220398, -0.5230907201766968, 0.05799699202179909, -0.18787150084972382, -1.4400451183319092, -0.4593656659126282, -0.18170157074928284, -0.7472847104072571, 2.6815273761749268, -1.3958606719970703, -0.9150403738021851, -0.8462086915969849, -1.2451791763305664, 0.8355597853660583, 1.532516598701477, -0.3438907563686371, 0.67596435546875, -0.6734820604324341, 0.20107941329479218, -0.5299105048179626, -2.5801215171813965, -0.6751617789268494, 0.8098520040512085, -0.18599195778369904, -1.2859299182891846, 1.7317826747894287, 0.010323558934032917, 1.0876299142837524, -0.8233328461647034, 1.1645312309265137, -0.8245142698287964, 1.2385599613189697, -0.06410884857177734, 0.08284416794776917, -0.006217596121132374, 0.46188557147979736, 0.5314555764198303, -0.08668649941682816, 0.04495031386613846, 1.5360815525054932, -0.2832295894622803, -0.5467908978462219, -0.6882690191268921, 0.7377444505691528, 0.44819799065589905, 1.2207016944885254, -0.548121988773346, 0.2290629893541336, 0.40704768896102905, -0.14370769262313843, -1.5342135429382324, 0.9565713405609131, -0.6254227757453918, 0.5923303961753845, -1.468493103981018, -0.4735952615737915, -1.1387399435043335, 1.3672689199447632, 0.9404526948928833, 2.4640448093414307, -0.7303180694580078, 0.6087000370025635, 2.693321466445923, 0.8161460757255554, -0.9784628748893738, -0.07864212989807129, -0.5293123126029968, 0.8902027010917664, 1.5835881233215332, -0.5141177773475647, -0.6584720015525818, 1.4710465669631958, -1.3996425867080688, -0.001983328489586711, 0.5144360661506653, 0.09475010633468628, 1.1243928670883179, 1.212438702583313, 0.6553507447242737, 0.7988481521606445, -0.05212694779038429, 0.4773359000682831, -0.2559024691581726, -0.7648729681968689, -0.24962058663368225, -1.3546656370162964, 1.0489404201507568, 0.3978717625141144, -0.022785719484090805, -1.2095346450805664, -0.8199531435966492, 1.6828464269638062, -0.3567168712615967, 0.07167267799377441, -0.7750047445297241, -0.022060267627239227, 0.7950064539909363, -0.8726743459701538, -0.28800690174102783, 0.8179494738578796, -1.6491026878356934, -0.39034172892570496, 0.9005075693130493, 1.4919302463531494, 0.27709755301475525, 0.349538117647171, -0.2876955270767212, -2.1123008728027344, 0.4392566680908203, 1.0767663717269897, 3.502035140991211, 0.9762413501739502, 0.741608738899231, 0.7056394219398499, 0.07593270391225815, 0.19369985163211823, 0.8173706531524658, 0.17530982196331024, 0.5848337411880493], [-0.09202349185943604, -0.04766370728611946, 1.41873300075531, -0.9084949493408203, 1.114486575126648, 0.059019818902015686, -0.015250897035002708, 1.172580361366272, 1.6078159809112549, 0.40636616945266724, -0.6257662177085876, 1.805827260017395, -0.25550758838653564, 0.6449736952781677, 1.24158775806427, 0.838308572769165, 0.18936829268932343, 0.1419493854045868, -1.451363205909729, 0.15381494164466858, 0.11538632959127426, 0.719581127166748, -0.6049603223800659, 1.93975031375885, 0.9235051870346069, -0.5710073709487915, -0.18719355762004852, -0.9272704720497131, 1.175049066543579, 1.2197343111038208, -0.9095941185951233, 0.7051082253456116, -2.2535834312438965, -0.02475312165915966, 2.4877586364746094, -1.1495673656463623, 0.5260288715362549, 0.780444324016571, 0.31628695130348206, 0.15015757083892822, 1.589648962020874, 0.5574895739555359, -0.5951926708221436, -0.34904760122299194, -0.1643742471933365, -1.5051864385604858, 1.092111587524414, -1.9900325536727905, -0.5695357918739319, 1.0331789255142212, 0.9372453689575195, 0.248403862118721, 0.6545270085334778, 0.19730816781520844, 1.233594536781311, 1.0504133701324463, 1.4123724699020386, 2.150500774383545, -1.0702685117721558, 1.5252269506454468, 0.44558003544807434, -0.9249035716056824, -0.11511434614658356, 0.22946801781654358, 1.037536859512329, -1.6437597274780273, -1.1008286476135254, 1.008072018623352, 1.4097880125045776, 0.701411247253418, 0.8199907541275024, -0.8850074410438538, -1.7009224891662598, 1.1728769540786743, -2.4082071781158447, -1.3277807235717773, 0.9160820245742798, -0.24895833432674408, -1.5925440788269043, -0.931493878364563, -0.39602935314178467, 0.5186716914176941, 0.18028581142425537, 1.104949712753296, 1.1603002548217773, -0.43704530596733093, -0.8183920979499817, 0.3135431408882141, 0.1061503067612648, 0.01364124845713377, -0.3208558261394501, 0.10615633428096771, 0.09871798753738403, 0.011941621080040932, -0.5306981801986694, 0.558711051940918, -1.2314194440841675, -2.101494789123535, 0.2504585385322571, 0.07080474495887756, -1.3791006803512573, -0.875196635723114, -0.42888495326042175, -0.10551141947507858, -1.098841905593872, -0.08422104269266129, 1.4077684879302979, -0.9680721759796143, -0.4691725969314575, 0.27891889214515686, 1.3462841510772705, 0.7296552062034607, 1.4476090669631958, 0.3570573627948761, 1.1196786165237427, 1.349833369255066, 0.16512560844421387, -0.5860408544540405, -1.8534932136535645, -0.3239521086215973, -0.6949576139450073, -1.797341227531433, -1.6113975048065186, -0.5355669260025024, 0.13291075825691223, 0.026064183562994003, 1.4356873035430908, -0.33198466897010803], [1.5903942584991455, 1.4900168180465698, 1.0566189289093018, 0.5744363069534302, -0.205438032746315, -0.09259586036205292, 0.4867076277732849, 0.4194486439228058, 0.6419752240180969, 0.47485318779945374, 2.104374408721924, 0.14560788869857788, -1.6274324655532837, -1.1198084354400635, 1.0476478338241577, -0.8499729633331299, -0.5012799501419067, 0.6099533438682556, 0.5048015117645264, -0.5379908084869385, 0.858180582523346, 0.5248367190361023, -1.3110443353652954, 1.9917762279510498, 0.9273934364318848, -0.5067535638809204, 1.382103681564331, 0.7925697565078735, -0.36926987767219543, 0.425550639629364, -1.0372259616851807, 1.5796281099319458, -0.3028489649295807, -0.17565597593784332, 0.12940111756324768, -1.3626165390014648, 0.5196892023086548, 0.3500863015651703, 1.0966547727584839, -1.537950038909912, -0.2450786530971527, -0.9191237688064575, -0.5162444710731506, -0.9951047897338867, 2.0120456218719482, 0.7594829201698303, -0.7828044891357422, 0.36736616492271423, -0.40386223793029785, -0.1778487265110016, -0.4902411997318268, -0.03893347457051277, 0.5165543556213379, -1.2022937536239624, 0.9663018584251404, -2.0230867862701416, -2.1822423934936523, -1.9597574472427368, 0.5585449934005737, -1.3571364879608154, -0.7062016725540161, 1.3711967468261719, -0.8830928206443787, -1.1721855401992798, 0.14627543091773987, -1.561729073524475, 1.0955708026885986, 0.08722628653049469, -1.3422634601593018, 1.6183819770812988, -0.6066936254501343, -1.1859372854232788, -0.04388491064310074, 0.8393107652664185, 1.5681699514389038, 0.48020610213279724, 0.14421826601028442, 1.1935701370239258, -1.7843172550201416, 1.4030815362930298, -0.729292094707489, -1.5120820999145508, -0.1925508826971054, 1.3809137344360352, -0.5931978225708008, 0.44995245337486267, -1.8177887201309204, 0.5057402849197388, 1.7175430059432983, -0.457563191652298, -0.27155429124832153, 1.0072193145751953, -0.9670859575271606, 1.1477022171020508, 0.49191147089004517, -0.43291065096855164, 0.3215700685977936, 0.9497870206832886, 0.7453525066375732, -0.9910749197006226, -0.41000986099243164, -1.362868309020996, -0.010069933719933033, 0.40697383880615234, 0.15602025389671326, 0.25731074810028076, 0.9494842886924744, -1.0499452352523804, -1.2276842594146729, -1.4258627891540527, 1.1487210988998413, 1.0334326028823853, 0.7300562262535095, -1.461911916732788, 2.0718841552734375, 1.175328254699707, -0.28744828701019287, 0.31360146403312683, 0.442624568939209, -1.3938465118408203, 0.5966806411743164, 0.5411987900733948, -1.2550389766693115, -0.5473033785820007, 0.049141060560941696, -0.8241487741470337, -0.3789631426334381, 0.14060306549072266], [-0.7518917322158813, 0.9210746884346008, 0.19499856233596802, 0.8657514452934265, -0.12500108778476715, 0.11640320718288422, 0.9561702013015747, -0.4010991156101227, 0.34931066632270813, 0.6326578855514526, 0.7825428247451782, -0.14833679795265198, 0.098692886531353, -0.469153493642807, -0.7470259070396423, 1.0603801012039185, 0.1534217745065689, -0.2373417615890503, 1.6745842695236206, 1.2427493333816528, -0.7550076246261597, 0.271304190158844, -1.4725953340530396, 1.7798343896865845, 1.0936975479125977, 0.22345338761806488, -0.3814174234867096, 2.2268455028533936, 0.5025995969772339, -0.8648614883422852, 0.9430757761001587, -1.6347806453704834, -0.4030658006668091, -0.586696207523346, -0.6523153781890869, 0.3744755685329437, 0.18478171527385712, 0.6764146089553833, -0.11165209859609604, 1.2334176301956177, 0.5565888285636902, 0.2671789526939392, 2.497119665145874, -1.0652114152908325, 1.8119540214538574, -0.7467882633209229, 0.10739995539188385, 1.7898049354553223, -0.8793987035751343, -1.164115309715271, 0.35033246874809265, 0.19758537411689758, 1.633601188659668, 1.1251434087753296, 0.11062787473201752, -1.1312689781188965, -0.1668599396944046, -0.8391612768173218, 0.42228105664253235, -0.4511169195175171, -0.42542916536331177, 0.06154516711831093, -0.796680748462677, 0.7866137623786926, 0.3142858147621155, -0.48506322503089905, 0.11863772571086884, 0.9910622835159302, -0.8957634568214417, 0.11203634738922119, 1.3855564594268799, 2.528214931488037, 1.0030713081359863, -1.088869333267212, 0.18595802783966064, -0.7877805233001709, -1.5601195096969604, 0.1512330174446106, -0.4223840832710266, -1.3692138195037842, 0.48504799604415894, 1.4483810663223267, 1.0313360691070557, 0.409124493598938, -0.5901015400886536, -0.8939590454101562, -0.5540136098861694, 0.2129725068807602, -1.0445680618286133, 2.1156394481658936, -0.26935043931007385, 0.7539967894554138, -0.29227951169013977, 0.6117953062057495, 0.11565303802490234, -0.049794070422649384, -1.351365089416504, 0.5156105756759644, 0.9365181922912598, -0.185980424284935, 0.6111329793930054, 0.5951264500617981, -0.9287815690040588, 0.4197794795036316, -1.3066630363464355, -0.36932432651519775, 0.9099190831184387, -1.5855495929718018, -0.21304164826869965, 0.19901655614376068, 0.6566834449768066, -0.46709251403808594, 1.5014331340789795, -0.2574670612812042, 1.28882896900177, 1.832903504371643, 2.5334761142730713, 0.7330537438392639, -0.1219152957201004, -0.03358970209956169, 0.18002460896968842, 2.4547030925750732, -0.851096510887146, 1.3924925327301025, 0.17391085624694824, 0.04069085791707039, 0.48564451932907104, 1.3884828090667725], [-0.7185223698616028, -0.10975229740142822, 0.5003179907798767, -1.0512831211090088, 0.5842300057411194, -0.28051674365997314, 0.6746966242790222, 2.321455478668213, -0.22611123323440552, 1.918437123298645, 0.9904688596725464, 0.5452885031700134, 0.19296786189079285, 0.8051660060882568, -1.3307271003723145, -0.7537273168563843, -1.6232904195785522, 1.4981411695480347, -0.981278121471405, -0.9672251343727112, -0.23595553636550903, -0.07859242707490921, 0.8186237812042236, 0.8362865447998047, -1.2440721988677979, -1.8942828178405762, -0.5809401869773865, 2.2764179706573486, -2.161102056503296, -0.38257071375846863, -0.5764288306236267, -0.31968429684638977, 0.10971404612064362, -0.008604073897004128, -0.4292491376399994, 0.6744655966758728, -0.4143376350402832, -0.2756979763507843, 0.7178928256034851, -0.2169192135334015, -0.5515736937522888, -1.4889274835586548, 0.011065466329455376, -0.9427066445350647, -0.6966431140899658, -0.24247701466083527, -1.2009004354476929, 0.5251671075820923, -3.0100138187408447, 0.6720764636993408, -1.5983190536499023, -1.508880853652954, 0.47930213809013367, 1.7021517753601074, 0.8142974376678467, 0.18825992941856384, -0.8057754635810852, -3.0404698848724365, 0.18514414131641388, 1.6033226251602173, -0.7617986798286438, 0.22084875404834747, 0.998749315738678, 0.4309185743331909, -0.7291759252548218, 0.5602255463600159, -0.07292235642671585, 0.2561185657978058, -1.2104592323303223, 0.7022861838340759, -0.6585028171539307, -0.46855294704437256, 0.2621687352657318, -0.045018892735242844, -0.07800734788179398, 1.2383325099945068, 1.7002087831497192, 0.31165972352027893, 0.43681713938713074, -0.45770731568336487, 1.5923947095870972, 0.1939605325460434, 0.6735012531280518, -1.3989295959472656, -0.7420294284820557, -0.12171482294797897, 0.7245028018951416, -0.7058582305908203, -2.3637518882751465, 0.1300664097070694, 0.7527573704719543, 1.3489267826080322, -1.8244500160217285, -0.2565573453903198, 1.1013846397399902, 0.5864878296852112, -0.5790593028068542, 0.3008767366409302, -1.8994563817977905, 0.18347260355949402, -0.8244454860687256, 0.35936057567596436, 0.22630269825458527, -0.05964818224310875, 1.0166488885879517, -2.2438724040985107, 0.7660902142524719, 1.514302134513855, 1.5222336053848267, 0.7884762287139893, 1.614651083946228, -0.09152156114578247, 0.6575332880020142, 0.7852333784103394, -2.0757334232330322, 0.22549179196357727, -0.5164425373077393, -0.8598861694335938, -1.8141242265701294, 2.2162013053894043, 0.9586371183395386, -0.22241589426994324, 0.8479713201522827, 0.21619868278503418, -1.5958877801895142, -0.31858909130096436, -1.4911999702453613, -0.9879659414291382], [-0.22926698625087738, 0.26576879620552063, -0.9770460724830627, -0.8058613538742065, 1.5514991283416748, -0.2535346746444702, 0.30044326186180115, 1.2559125423431396, 1.4135688543319702, 1.1324036121368408, -1.2346612215042114, -0.8533157706260681, -0.9530947804450989, 1.582907795906067, 1.2410914897918701, 1.3343901634216309, 1.338321328163147, -0.9566812515258789, 1.2548918724060059, 0.058459796011447906, 1.2122770547866821, -2.447451591491699, -0.724055290222168, -0.2798381447792053, -0.07567849010229111, -0.6031134128570557, -0.3741656243801117, 0.28539538383483887, -0.6501697301864624, -2.392103910446167, -1.17593514919281, -1.2672853469848633, 0.08693437278270721, 1.2598713636398315, 0.24791860580444336, 0.7532559633255005, 0.48743072152137756, -0.4819101095199585, -0.45714959502220154, 0.8401210308074951, 0.07718914747238159, 1.0661646127700806, 1.0728962421417236, 0.5681394338607788, -0.7777152061462402, 0.8081772923469543, -0.28199148178100586, 1.2770966291427612, -1.460924744606018, 1.5739929676055908, 0.34626272320747375, 1.0687007904052734, 0.27875611186027527, -0.31823334097862244, 0.4850054979324341, -0.6129570603370667, -0.9539574980735779, -0.9962078928947449, -2.1369917392730713, -0.8318299651145935, 1.893736720085144, -2.4774892330169678, -1.1987289190292358, 0.6292533278465271, 0.6290555000305176, -0.5923904776573181, -0.2981867790222168, 1.5141764879226685, -1.2284824848175049, -0.38823801279067993, 0.7634198069572449, 0.9889682531356812, 0.32580843567848206, 0.03170550987124443, -1.4545934200286865, -0.11359179764986038, -1.3062089681625366, 0.37705880403518677, -1.3388080596923828, 0.8278568387031555, 1.503798007965088, 1.6432589292526245, 1.1020103693008423, -0.05006997659802437, 0.26306334137916565, -0.524350106716156, -0.8913697004318237, 0.5973702073097229, 0.12496118247509003, 0.6845517754554749, 0.3509814441204071, 1.0291491746902466, -0.506675660610199, -0.08772461861371994, -1.0670878887176514, -0.6326815485954285, -0.6521430015563965, -0.8303872346878052, 0.7651357650756836, -0.8132141828536987, -0.7946163415908813, -1.5918351411819458, 0.4132835566997528, -2.5190200805664062, -0.4281410872936249, 0.9230989217758179, 0.6707973480224609, 0.7736412286758423, 0.6079687476158142, 1.178780436515808, -0.9495458602905273, -0.09101822227239609, -0.1687919646501541, -0.4415953755378723, -0.6056520938873291, 0.028814053162932396, 0.26236817240715027, 0.9879337549209595, -0.7153169512748718, -0.3673912584781647, -2.231806993484497, 1.8512595891952515, -0.3866112232208252, -0.11159133166074753, -1.0271002054214478, -0.010666904971003532, -2.096496343612671, -0.5191658735275269], [-0.2953221797943115, -1.0696617364883423, -1.6620001792907715, 0.8303449153900146, -0.8993768095970154, 0.6082672476768494, 0.2569587528705597, -2.292165517807007, -2.3788340091705322, -0.34327980875968933, -0.928226113319397, -0.030967650935053825, -0.8433026671409607, 0.9026267528533936, 1.5717864036560059, -1.0850038528442383, -0.4196927845478058, 2.373236894607544, 0.6186770796775818, -1.6381508111953735, 0.7366304993629456, -0.8013069033622742, -1.3043498992919922, -0.6264142394065857, 0.8735829591751099, 0.6763916611671448, -0.6976483464241028, -0.13718947768211365, 0.8328666687011719, -0.40633347630500793, -0.4992671310901642, 0.3423343002796173, 0.785515546798706, 0.027916885912418365, -1.3008079528808594, 0.6406434178352356, -0.5595689415931702, -0.848585844039917, -0.3262525796890259, -0.9910727739334106, -1.6005501747131348, -0.4274330735206604, 0.7722243070602417, 2.0637106895446777, -0.40951037406921387, 0.14862164855003357, 0.8706102967262268, -0.5213263034820557, 0.025123709812760353, 1.0173906087875366, -0.8336016535758972, -1.6735167503356934, -1.6201974153518677, 1.647637128829956, -0.11291006952524185, 0.0065787178464233875, -0.07867616415023804, 1.9948234558105469, 0.5193201303482056, -1.836928129196167, 1.0268975496292114, -1.1026662588119507, -1.9341049194335938, -1.3335455656051636, 0.23655354976654053, 0.4553055763244629, -0.6538986563682556, 0.04911283776164055, 1.169878602027893, 0.15773330628871918, -0.4074617326259613, -0.9083644151687622, -0.7088786959648132, -0.9072582721710205, 0.21972781419754028, -0.9709132313728333, -0.27260667085647583, -0.49151960015296936, -0.9406578540802002, 0.80628502368927, 0.47440066933631897, 0.31413987278938293, -1.113857388496399, 0.01771152764558792, -1.872890830039978, -1.1983054876327515, -1.2945175170898438, -0.6676816344261169, 0.4430968165397644, 0.1822706162929535, -0.19962657988071442, -1.1212882995605469, -0.13193733990192413, -1.9584791660308838, 1.5104202032089233, -0.2526598870754242, 0.25989478826522827, -0.467275470495224, 0.819705069065094, 0.7186668515205383, 0.3803698718547821, -0.024744706228375435, -3.2535409927368164, 0.497211217880249, -2.566896438598633, 1.654254674911499, -0.8787355422973633, -0.708268404006958, -1.0977269411087036, -0.6216430068016052, -0.5852319002151489, 0.5553910136222839, -0.47746339440345764, -0.3318956792354584, -1.2245539426803589, 0.02193758450448513, -0.12618613243103027, 0.8833476305007935, 0.4772108495235443, -0.2195216715335846, -0.05663658306002617, 0.3583395481109619, 2.0447068214416504, -2.4648923873901367, -0.517694890499115, -1.0328980684280396, -0.7295047044754028, -0.044844359159469604], [0.16378791630268097, -0.4584110975265503, 0.16754025220870972, -0.44355326890945435, -0.9233525991439819, -1.2761282920837402, 1.2567304372787476, 0.213611900806427, -0.6895360350608826, -1.1253657341003418, -2.2933170795440674, 0.06534866988658905, 1.1466916799545288, 0.1887618750333786, -2.304779052734375, -0.695270836353302, -0.011158107779920101, 0.16161029040813446, 0.7768205404281616, -0.5404585003852844, 1.1590418815612793, -0.5927801132202148, -0.9094085693359375, -1.1196104288101196, 0.25957930088043213, 0.10720714926719666, -0.605186402797699, 0.4501955211162567, -0.5428663492202759, -0.10511258244514465, -0.5458469986915588, 0.7530606389045715, -0.5070696473121643, -0.6197211742401123, 0.9286562204360962, 0.6022870540618896, -1.9572595357894897, -0.45796456933021545, 0.46299436688423157, -0.1533975452184677, 0.4150468409061432, 0.02607031911611557, 0.48294514417648315, 1.4121016263961792, -1.0463210344314575, 0.2636640667915344, -0.4178420603275299, 1.8874576091766357, -0.6053076386451721, 0.17979265749454498, 1.3910290002822876, -1.187385082244873, 0.6092238426208496, -1.000664472579956, 0.5387405753135681, -0.48858344554901123, 0.37234795093536377, -0.49949297308921814, -0.8962505459785461, 0.08865074068307877, -1.0249325037002563, 1.0615777969360352, -0.3221290707588196, -0.8725204467773438, 0.4009345769882202, -0.37729158997535706, -0.13091714680194855, -1.3513668775558472, -0.2911394536495209, -0.3336041271686554, -0.9428212642669678, 0.40424835681915283, 0.12357375025749207, -0.9580291509628296, 0.5204582810401917, 0.19663335382938385, 0.30360743403434753, -0.3766286075115204, -0.045833759009838104, 0.2292884886264801, 0.12144719064235687, 0.021787187084555626, -0.730958878993988, 0.30475014448165894, -0.9186515808105469, 1.3215008974075317, -0.6077376008033752, 1.2585039138793945, 0.8864995241165161, 0.2569499611854553, -0.08022081851959229, -0.34485745429992676, -1.0665173530578613, -0.7294355630874634, -0.20160824060440063, -2.0551705360412598, 1.2423244714736938, 1.8828336000442505, -0.11663582921028137, 0.3442809283733368, 0.49424684047698975, -0.9955662488937378, -1.0289268493652344, -1.2224299907684326, -1.9398040771484375, 0.4572139084339142, -0.931504487991333, 1.3424383401870728, 1.826429843902588, -0.7319933176040649, 0.24476562440395355, -0.6402842402458191, 0.9986189603805542, -0.42373859882354736, -0.840071439743042, 1.4046707153320312, 0.9253146052360535, 0.4384188652038574, 0.1967240273952484, -0.43917810916900635, 0.21962983906269073, 2.6780283451080322, -0.2893528640270233, 1.1525707244873047, 0.6881822347640991, 0.27143797278404236, 0.3291761875152588, -0.32642972469329834], [-0.34505102038383484, -0.003037462942302227, 0.4883746802806854, -0.29482555389404297, 0.2829018831253052, 1.1283330917358398, -0.8039456605911255, -0.4763420820236206, -0.39625540375709534, -0.28293150663375854, 0.1839992254972458, 1.8108890056610107, -0.0033615026623010635, 1.1481378078460693, 0.6540058255195618, 0.43931323289871216, 0.824210524559021, -1.6664149761199951, 1.0536845922470093, -1.7575026750564575, 1.002339243888855, -2.0430212020874023, 2.0096421241760254, 1.0243821144104004, 0.5399798154830933, -0.3958326280117035, 2.1131837368011475, 0.4902368485927582, -1.4510527849197388, -0.8700461983680725, -0.40200749039649963, -0.8579777479171753, -1.8162506818771362, 0.3406946659088135, 0.09124022722244263, -0.7706325650215149, -0.030118733644485474, 1.881489872932434, -0.5338639616966248, -0.5510729551315308, 0.308960497379303, -1.0604729652404785, -0.4152681231498718, -1.3505783081054688, -0.7769007086753845, 0.6345183849334717, -0.950559675693512, 1.003920316696167, -0.14898879826068878, 1.33234703540802, -0.5981993675231934, -0.4060220718383789, 0.55247563123703, 0.00956935714930296, 1.3992925882339478, 0.19697077572345734, 0.10228358954191208, 1.8792436122894287, -0.48035356402397156, 1.9100158214569092, -0.66762375831604, 0.9942111968994141, -1.3983060121536255, -0.7083866000175476, -0.6139249801635742, -0.8335482478141785, -1.5835940837860107, 0.9893139600753784, -1.9328312873840332, -0.6732762455940247, -0.9071871042251587, -1.4480559825897217, 0.5475286841392517, 0.5089933276176453, 0.25379931926727295, 1.7462413311004639, -0.06148763373494148, 1.7557286024093628, -1.1666587591171265, -0.5782493948936462, -0.8182551860809326, 2.8453168869018555, -0.0262156929820776, 0.20684000849723816, 1.3438259363174438, 0.4479886293411255, 0.627325177192688, -0.04076271876692772, 0.22800925374031067, -1.7477229833602905, 0.6662495136260986, -1.5144054889678955, 0.42478808760643005, -0.5468178987503052, -1.1803579330444336, 0.439435213804245, -0.6046075224876404, 0.9254411458969116, -1.6204252243041992, -0.2437065690755844, 0.12153754383325577, 0.46653762459754944, -0.44668474793434143, -0.14539207518100739, -0.19088204205036163, 0.4157091975212097, 0.10386735945940018, -0.850509762763977, -1.1152215003967285, 0.4579741358757019, 0.6481136679649353, 1.3113114833831787, -0.5059801936149597, -0.09349023550748825, 0.17236506938934326, -0.7130045294761658, -0.4656654894351959, -0.18496324121952057, 1.6302721500396729, 0.1957959532737732, 0.8352944850921631, 0.7150586843490601, -0.8761134743690491, -0.985167920589447, 1.728888988494873, -0.10546212643384933, -1.0524660348892212, -0.9482658505439758], [1.3474721908569336, -0.8759922385215759, -0.7798674702644348, -1.0147653818130493, -0.6373092532157898, 1.7868459224700928, 1.1993319988250732, -0.36731916666030884, -1.195749282836914, -0.7484087944030762, 1.3968236446380615, 0.07657185196876526, 1.1571147441864014, -0.20060735940933228, 0.16612184047698975, -0.04969331994652748, 2.6935877799987793, -0.6765444874763489, -0.2901894748210907, -0.3955981135368347, 1.943154215812683, 1.4150797128677368, -0.32786446809768677, -0.9924272298812866, -0.5603196620941162, 0.6028696298599243, 0.5525436401367188, 1.1147310733795166, 1.3268190622329712, -0.2430618554353714, -1.3113772869110107, -0.20794253051280975, 0.6937062740325928, -1.2670931816101074, -0.10688843578100204, 2.4285967350006104, -0.007833759300410748, -0.7988967299461365, 1.0670654773712158, -1.1054116487503052, -2.178816080093384, -0.050917696207761765, 2.0544865131378174, -0.47317588329315186, 1.6714026927947998, 1.5310447216033936, 1.3874603509902954, -1.1399619579315186, 0.046359967440366745, -0.9368552565574646, -0.4329649806022644, 0.8719385266304016, 0.672216534614563, 1.9977551698684692, -1.062576413154602, 1.406820297241211, 0.8107001781463623, -0.950905978679657, -1.0707602500915527, 0.7604035139083862, 0.570115327835083, 0.3973119556903839, 0.6034833788871765, -0.41222238540649414, -1.4661411046981812, 0.9933447241783142, 1.4650448560714722, 1.535648226737976, 0.1180860847234726, -0.4075597822666168, 0.6985813975334167, 0.5692253112792969, -0.0713668093085289, -0.1629633754491806, 0.5834647417068481, -1.0149675607681274, 0.8066065311431885, 0.6084391474723816, -0.9297232627868652, 0.5619082450866699, -0.3491954505443573, -0.41262269020080566, 1.2298718690872192, 0.3314247727394104, -1.3635077476501465, 0.7467169761657715, 0.5247035026550293, -0.4139685332775116, 1.716849446296692, 0.8703598380088806, 6.284617120400071e-05, -1.2186782360076904, -0.6240766048431396, 0.5336838364601135, -3.3652706146240234, 0.3919963240623474, 0.5580869317054749, 1.249700903892517, 0.13532160222530365, 0.026446567848324776, -0.8858373761177063, 1.6967703104019165, 0.08616732805967331, 0.7040128111839294, -0.8285614252090454, 1.118277907371521, 0.3988840878009796, 0.544641375541687, 0.9754824638366699, 0.27724024653434753, 0.37250879406929016, 0.07574715465307236, 0.6382827162742615, 0.14608077704906464, -0.9482481479644775, 0.15281552076339722, 0.4041568636894226, -1.1466686725616455, -1.7949713468551636, -2.5566906929016113, 0.006566207390278578, -0.4941144287586212, -0.44096022844314575, 1.3417762517929077, 2.235830545425415, -1.9675780534744263, 2.3930656909942627, 0.04707964509725571]], [[-0.8420848846435547, -0.09758614748716354, 1.2607827186584473, -0.8873891830444336, 0.8984737992286682, 0.0084818871691823, -2.213167428970337, -0.16130894422531128, 2.546891689300537, -0.6623049974441528, 0.28400981426239014, -0.1367419958114624, 0.7697848081588745, -0.7753949165344238, -0.30699801445007324, -0.8405590057373047, -0.597633421421051, -0.70423424243927, 0.5244295597076416, 1.9064335823059082, 2.2764527797698975, -0.5465953946113586, -0.6608413457870483, 0.8844273686408997, 0.5874342918395996, 0.5502797365188599, 0.1427326500415802, -0.4891047179698944, 0.08562097698450089, -0.20481565594673157, 0.17535719275474548, 0.4210796058177948, 0.6031660437583923, -0.8092361092567444, 0.058742985129356384, -0.3230331540107727, -0.1289302110671997, 1.0963318347930908, 1.2947311401367188, -0.8426129221916199, -0.09068700671195984, -0.8092560172080994, -1.812424659729004, 1.1282132863998413, 1.0298528671264648, -1.9739185571670532, -0.2306968718767166, -0.43084442615509033, -1.2182576656341553, -1.5916160345077515, -0.04006842523813248, -0.33833929896354675, -0.44975998997688293, -0.8732576966285706, 1.6009571552276611, 1.049407958984375, -2.0680458545684814, 1.618605136871338, -0.47278061509132385, 0.3665520250797272, -0.30472373962402344, -1.25692880153656, -1.224169373512268, -1.0926587581634521, 0.8015680909156799, -0.7539668679237366, -1.7022271156311035, 1.0045278072357178, 0.4811982810497284, 1.7861480712890625, 1.0288970470428467, -0.26786378026008606, -0.44660937786102295, -0.0991138219833374, -0.3867066204547882, -1.4508912563323975, 0.992963969707489, -1.462127447128296, 0.9082863330841064, 1.1262218952178955, 1.265712857246399, -0.9873963594436646, 1.8553614616394043, 0.5987741351127625, 1.149707317352295, 1.1013543605804443, -1.2175564765930176, 0.5352296233177185, 1.5565245151519775, -1.8035862445831299, -0.2753455936908722, -0.8112193942070007, 0.475579172372818, 0.2638776898384094, 0.06660578399896622, -0.16554147005081177, -0.2599821984767914, -0.7564226388931274, -0.17710484564304352, -0.9905958771705627, -1.2642114162445068, -0.6800066232681274, -0.6585279107093811, -1.6619058847427368, 0.785428524017334, -2.084975242614746, 1.0211600065231323, -0.23686452209949493, 0.7529661655426025, 0.9153985977172852, 0.8031628727912903, -0.9532154202461243, -0.47939497232437134, -0.8804871439933777, 1.8683680295944214, -1.7674237489700317, 0.015239981934428215, 0.7173215746879578, 0.6598366498947144, -0.817791759967804, 0.7250207662582397, -1.210191249847412, -1.0414565801620483, 0.11044217646121979, -0.041022494435310364, 0.689744770526886, -0.05835268646478653, 0.06921112537384033], [-1.7698240280151367, -0.6002169847488403, -0.3099432587623596, -0.9189499616622925, -0.909144937992096, -0.8307490944862366, 1.2932591438293457, 1.6578941345214844, -1.7181050777435303, 1.2494608163833618, -0.32819774746894836, -0.07276920229196548, -0.863496720790863, 0.9310077428817749, 0.40589964389801025, 1.2422137260437012, -1.4870346784591675, -1.615709900856018, -0.7473857402801514, 0.544443666934967, 0.7801890969276428, -0.23611527681350708, -0.8058974146842957, -0.2778919041156769, -1.7369720935821533, -1.4776701927185059, 0.6786106824874878, -0.7073526978492737, 0.8514596223831177, -0.20987114310264587, 0.280300110578537, 1.5029774904251099, -0.024470586329698563, -1.8647098541259766, 0.23274531960487366, 0.0209425650537014, -1.1868940591812134, -0.21490001678466797, -0.5301893949508667, 1.5065141916275024, -0.5360124707221985, -0.6662110686302185, -0.9186500906944275, -1.121322751045227, -0.8182168006896973, 0.2826938033103943, -1.1239500045776367, -0.6977444291114807, 0.24906344711780548, 2.3032643795013428, 1.5798215866088867, 0.2063549906015396, -0.9783197641372681, -0.3038780093193054, 0.128486767411232, -0.7099384665489197, 0.7359831929206848, -0.07412280887365341, -1.155791997909546, -1.5274839401245117, 0.40253862738609314, -0.29749637842178345, 0.38843151926994324, -1.2594565153121948, -0.19984124600887299, 0.8908401131629944, -0.2211000919342041, -0.657515823841095, 1.5882489681243896, 0.4908711612224579, 0.6301643252372742, 2.285417318344116, -0.8438712358474731, -0.38906270265579224, -0.5925664305686951, -0.4609789252281189, 0.4464460611343384, 0.6482523083686829, 0.1265815645456314, 0.4634813964366913, -0.9072567820549011, 0.14294041693210602, -3.1491806507110596, 0.9859470725059509, -0.015206851996481419, 0.6098744869232178, -1.209574818611145, 0.4366934895515442, -0.8733066320419312, -0.30624374747276306, -0.8987082839012146, 0.33983755111694336, 1.1314610242843628, -0.8071130514144897, 1.2970143556594849, 1.3138363361358643, -0.8192815780639648, -0.05006493255496025, 0.8870099782943726, -1.0987590551376343, -0.840582549571991, 0.7097324132919312, -0.4517649710178375, 1.0342929363250732, 1.0197170972824097, -0.38414865732192993, 0.4348060190677643, -2.2234857082366943, 1.0176432132720947, -1.1873095035552979, -1.2917178869247437, -0.31421685218811035, 0.7534536123275757, 0.9492095112800598, -0.9043976664543152, -1.5086919069290161, 0.04243521764874458, -1.0805981159210205, -0.06463992595672607, 0.31552088260650635, -1.9013196229934692, 0.19416724145412445, -0.5414760112762451, -0.09997083991765976, 1.1921048164367676, -0.8131222724914551, 0.6327794194221497, 0.6764262318611145], [1.3880037069320679, -0.7286482453346252, 0.08478138595819473, -3.1630327701568604, 0.17347924411296844, 0.13074550032615662, -0.4813208281993866, -0.03680814057588577, -0.12719424068927765, -0.48812365531921387, 0.4592973589897156, -0.5910556316375732, -1.112735390663147, -0.3833763301372528, 0.060499630868434906, -0.2243874967098236, 0.9270383715629578, -0.31359443068504333, -0.6073119044303894, 0.736541748046875, 0.1992097645998001, -1.2217226028442383, 0.3368937373161316, -1.1065407991409302, -0.7077690362930298, 1.8728477954864502, 1.526082992553711, -1.538343071937561, 1.6703104972839355, -0.017450137063860893, -1.6155942678451538, 1.4665498733520508, -0.5038803219795227, 1.7004303932189941, 0.23409239947795868, 0.4871007204055786, -0.17907527089118958, -0.5249640941619873, 0.48502418398857117, -0.8218110203742981, 0.22552669048309326, -0.6355027556419373, 0.6643446087837219, 0.5136303901672363, -1.0184149742126465, -0.6242307424545288, 0.720434308052063, 0.5414584875106812, 0.890593409538269, -1.435341715812683, 1.230211853981018, 0.5342888832092285, -0.21113519370555878, 0.9461296796798706, -2.1832244396209717, 0.5974839925765991, 0.2468605935573578, -1.5421009063720703, 1.2194750308990479, -0.9994413256645203, 0.1776605248451233, 0.12846717238426208, 1.074323296546936, 1.4307702779769897, 0.7070430517196655, -0.287557989358902, -0.5944305062294006, 0.1338784545660019, -0.097408726811409, 0.4017373323440552, 1.429880976676941, -0.0918252170085907, 0.26557034254074097, 0.06256581097841263, 0.10472260415554047, 0.2774161100387573, -0.7124656438827515, 0.5021542906761169, 0.38215944170951843, -0.0980897918343544, -1.617550253868103, 0.006369838956743479, 0.515038251876831, -1.2044636011123657, -0.6191007494926453, -0.1402752697467804, 0.8944921493530273, -0.6775226593017578, -0.2857552766799927, 0.5730380415916443, 0.8984712958335876, 0.12162736058235168, 0.049248840659856796, 0.38914015889167786, 0.284706175327301, 0.07615168392658234, -1.771990180015564, 0.32260408997535706, -0.1685762256383896, -1.110885739326477, 1.4333223104476929, 0.3331683278083801, 2.0296225547790527, -0.2528766393661499, -1.1539843082427979, 0.4282091557979584, -0.4165807366371155, 1.89883291721344, 0.0503203310072422, 0.7634414434432983, -0.6250267624855042, 1.0820157527923584, 1.6562381982803345, -0.8226628303527832, -1.6098606586456299, 0.13288253545761108, 1.8232421875, 1.2247073650360107, -0.27963578701019287, 1.877388596534729, -0.5578170418739319, -0.27513259649276733, -0.19490176439285278, 1.191343069076538, -0.5205691456794739, 1.3777198791503906, 1.213720679283142, 0.41452065110206604], [0.4731965959072113, -1.0867650508880615, 0.36529213190078735, 1.2230587005615234, -0.8212100863456726, -0.9514446258544922, 1.7744332551956177, -0.3029860854148865, 0.9698521494865417, -0.1874585598707199, -0.3193581700325012, 0.5621806383132935, 0.5212012529373169, 0.5472068190574646, 0.6160128116607666, 0.23240084946155548, 0.5407387018203735, 1.7769430875778198, -1.3889245986938477, -1.2170113325119019, 0.17120087146759033, -0.7630832195281982, 1.7133821249008179, 1.4465715885162354, -0.6811662316322327, 0.001196400378830731, -1.0045517683029175, 0.5115078091621399, 0.7017806172370911, 0.35486242175102234, 0.2165067493915558, 0.8943942189216614, 0.007773916237056255, -0.30160972476005554, -0.9171001315116882, -0.3951599597930908, -0.8033998012542725, -1.2398895025253296, -0.850562572479248, 0.06511908024549484, 0.8644833564758301, -0.3006924092769623, -0.6139533519744873, -1.7486757040023804, 0.018716778606176376, 0.6583198308944702, 0.08877431601285934, -0.5339060425758362, 0.9999680519104004, 0.26368531584739685, 0.6157394051551819, -1.5826740264892578, -0.1294233202934265, 1.3273102045059204, 0.969474196434021, 0.6250964999198914, 1.6388890743255615, -0.8084178566932678, 1.9334913492202759, 0.8810744881629944, 0.06825607270002365, 1.3148423433303833, 0.018147069960832596, 0.12215610593557358, 0.005975441075861454, 0.527825653553009, 0.635041356086731, 0.06202269718050957, 0.24152915179729462, 0.5946447253227234, -0.16333536803722382, 0.24498476088047028, -1.7062103748321533, -1.406398892402649, 1.616599440574646, -0.19396834075450897, 0.4776898920536041, 0.533905565738678, -0.2615944743156433, 0.49535703659057617, 1.814194917678833, -0.5300115346908569, -1.4167850017547607, 1.3338030576705933, -0.34206998348236084, -0.7048787474632263, -0.7503555417060852, 0.26601144671440125, -1.2506474256515503, 1.266489863395691, 1.8448352813720703, -1.612960696220398, -0.49676525592803955, 0.5460910797119141, 2.6959290504455566, 1.6105682849884033, 0.9408723711967468, 0.04978606477379799, -2.2492592334747314, 2.3666481971740723, -2.195542097091675, -1.171596884727478, -2.755413055419922, 1.7388278245925903, -2.1433591842651367, -1.7899402379989624, -0.3255421817302704, 1.3062342405319214, 0.25757014751434326, 0.1874886304140091, 0.948116660118103, -0.191525399684906, 0.17745676636695862, -0.7424665689468384, 1.5246679782867432, -0.5643502473831177, -0.04720291495323181, 0.7721895575523376, -1.5148752927780151, -1.2196979522705078, 1.5450630187988281, -0.7197016477584839, 0.14107179641723633, -0.811038613319397, 1.4524004459381104, 0.11344946175813675, -0.38648471236228943, 0.15720854699611664], [0.15688583254814148, 0.8676983714103699, 0.7379401922225952, -0.6508821845054626, 1.499076247215271, -0.33715707063674927, 0.40157076716423035, -1.637588381767273, -0.4105222523212433, 0.7123782634735107, -1.811692476272583, 0.45347920060157776, -1.3672672510147095, 0.9011720418930054, 0.3513924479484558, -0.13238105177879333, 1.9090362787246704, 0.8582288026809692, 0.41413021087646484, -0.8323530554771423, 0.49028778076171875, -0.5662314295768738, 1.8326700925827026, 1.2693021297454834, -0.06841444224119186, -1.0353525876998901, -0.08622883260250092, 1.1646218299865723, 0.29491326212882996, 2.360954999923706, -0.5825190544128418, -0.3681677281856537, -0.59662264585495, 0.8431475162506104, 0.925986647605896, -0.7683084607124329, -0.06585367023944855, -1.0682814121246338, 0.4010612964630127, 1.1717936992645264, 0.9272884726524353, 0.11176304519176483, -0.6846615076065063, 0.259822279214859, 2.3714540004730225, 0.5392730832099915, -1.5813024044036865, -0.2465391457080841, -0.4041222631931305, -0.6293182373046875, -1.3121556043624878, 0.5258233547210693, -0.22791200876235962, -2.3816726207733154, 0.716383159160614, -2.0518136024475098, -0.2141348421573639, 0.8086969256401062, -0.04604475572705269, -0.8540214896202087, -1.2725025415420532, -0.7528305649757385, 0.5203066468238831, 0.06953626871109009, -0.3171895146369934, -1.9904670715332031, -0.5838937163352966, 0.5157138705253601, -0.4502476453781128, 0.5973765254020691, -1.5693634748458862, 1.4501713514328003, 1.894089698791504, -0.5419800877571106, -0.20154337584972382, -0.5338770151138306, 0.3383726179599762, 0.5663775205612183, 0.18268448114395142, 1.956101417541504, 0.32315897941589355, -0.30113527178764343, -0.8535177111625671, 1.0723251104354858, 0.5354126691818237, 0.07027027010917664, 0.09766378253698349, 0.9133367538452148, 1.8948934078216553, 1.495662808418274, -1.0006626844406128, -0.340192973613739, -0.008943283930420876, -0.5249000787734985, 1.388683557510376, 1.9008535146713257, -0.3174939453601837, 0.5137031078338623, 0.3942011594772339, -1.501886248588562, -0.32864347100257874, -0.6497868895530701, 0.08298242092132568, 0.6034612059593201, -0.1781555563211441, -1.0567907094955444, 0.5928325653076172, 1.2378697395324707, -1.602487325668335, -0.14381958544254303, -0.14128391444683075, 0.9893893003463745, 1.7476688623428345, 0.1572369784116745, 0.06242500990629196, 0.2605006694793701, -0.8901237845420837, 0.45395076274871826, 1.8904492855072021, 1.237243890762329, 0.4760359227657318, 0.7450711131095886, -0.3840675950050354, 1.255723476409912, 0.4980933964252472, 1.619434118270874, -1.3619709014892578, 0.7836014628410339], [-0.5929936170578003, -0.4086473882198334, -1.1973369121551514, 0.18272295594215393, 1.8450793027877808, -0.2116188257932663, 0.35257983207702637, -0.303671658039093, -1.1036367416381836, 0.27021268010139465, 2.0589354038238525, 2.7936272621154785, 1.2840933799743652, 0.09085545688867569, -0.9535512924194336, 0.22453010082244873, -0.26043790578842163, 1.1264207363128662, 0.8771411776542664, -1.7139091491699219, -0.7695232629776001, 0.9719972610473633, 0.5417147278785706, -0.6752691268920898, -1.157676100730896, 0.41181832551956177, -1.008621335029602, -0.35047852993011475, -1.0609527826309204, 0.022482438012957573, 0.8868164420127869, -0.47658947110176086, 2.286106824874878, -1.4638465642929077, -0.9280365109443665, 0.012581928633153439, 0.24325555562973022, -1.507981777191162, 0.10401810705661774, 0.929565966129303, -1.2707695960998535, -0.4098793864250183, -0.8057934641838074, 0.5486623644828796, -0.9728558659553528, 0.8088281154632568, 0.5139827728271484, 1.6533328294754028, -0.8114999532699585, -0.6563020348548889, -0.8743404150009155, 0.1640995740890503, -0.5815942883491516, -1.4269936084747314, -0.8927829265594482, 1.2544071674346924, 0.827164888381958, -1.2951761484146118, 0.8140358924865723, 1.8936457633972168, 0.021472038701176643, -1.0773868560791016, 1.5773649215698242, 1.636350393295288, -0.5358284711837769, -1.3603395223617554, -1.1595499515533447, 1.4484233856201172, 0.7984007000923157, 0.41014033555984497, -1.299822449684143, 1.1489028930664062, -1.5943254232406616, 0.10039142519235611, -1.1696622371673584, -0.37257200479507446, 0.7116169333457947, -1.0479869842529297, 1.8610011339187622, 0.38091278076171875, -0.40573441982269287, 0.06871002167463303, 0.5284467339515686, 1.2028839588165283, -0.5604866743087769, 1.1695681810379028, -1.4045593738555908, 1.2842061519622803, -0.1360769271850586, 0.2921653091907501, 0.2510376274585724, -0.16721892356872559, 1.265357255935669, -1.9981331825256348, -1.4138858318328857, -0.11185789108276367, 0.05700811743736267, -0.802988588809967, 1.3478705883026123, -0.9874293208122253, -0.6767877340316772, -0.41551434993743896, -0.47746872901916504, -0.9758047461509705, -0.47397372126579285, -0.21056273579597473, 0.4355272650718689, -0.8015840649604797, -0.5527912974357605, 0.15298990905284882, 0.025336917489767075, -1.237730860710144, 0.06307942420244217, -0.8325855731964111, -1.7948330640792847, -1.9253968000411987, 1.4686378240585327, 0.25529035925865173, -0.8753595352172852, -1.488128900527954, -1.7767430543899536, -0.5083788633346558, 0.8854390978813171, -0.872118353843689, -1.8014417886734009, 1.3408340215682983, 0.814347505569458, 0.9648537039756775], [-0.11052977293729782, 0.7790848612785339, -1.1878917217254639, -1.4519544839859009, -0.7725050449371338, -0.33597904443740845, -0.0420980341732502, -0.3004678189754486, -0.998908519744873, -1.5338971614837646, -0.3258686661720276, -2.1590824127197266, 0.15752550959587097, -0.3792541027069092, -2.4851231575012207, 0.7415242791175842, -0.16968762874603271, -0.4673304557800293, 0.0779290422797203, 0.7149803042411804, 0.5112883448600769, 0.6386305093765259, -0.0926642194390297, 0.835056483745575, -0.8058256506919861, -0.3747568130493164, -0.11548162251710892, -0.7932275533676147, 0.5059810280799866, 1.3944520950317383, -1.535732388496399, -0.20466117560863495, 0.813981831073761, 0.9837263226509094, 1.42547607421875, -0.7665818333625793, 0.30361396074295044, 0.3773733973503113, -1.3642165660858154, -2.038102626800537, 0.37132135033607483, -1.2922488451004028, 0.09501030296087265, -1.781113624572754, -0.3654947876930237, 1.1893292665481567, -0.7194201946258545, -0.3439497947692871, 0.02385762706398964, -1.5165445804595947, 1.5440959930419922, 1.6431249380111694, 0.6312540173530579, 0.07950767874717712, 0.24947631359100342, 0.943640947341919, -0.5331088900566101, 1.2523853778839111, 1.3411610126495361, 1.1928372383117676, 0.06231299042701721, -1.7093337774276733, -0.6874746084213257, 0.029658610001206398, 0.37355276942253113, -0.17765358090400696, -1.1043360233306885, -0.8822921514511108, 0.015971390530467033, -1.7288854122161865, 0.689810037612915, -1.0021957159042358, 0.2536175847053528, 0.2344314008951187, 0.031845804303884506, 1.3530622720718384, 0.6716488599777222, -1.4826812744140625, 1.7379416227340698, -1.4199084043502808, -0.1518356055021286, 1.0281929969787598, 2.0808119773864746, 0.6220405697822571, -0.7145789265632629, 0.2686893045902252, 0.36865201592445374, -0.6488391757011414, 1.6343698501586914, -0.3956291675567627, -0.9513610601425171, 0.42468008399009705, 1.9862140417099, -0.7468408942222595, 0.31751543283462524, -0.4721819758415222, -0.20152971148490906, -1.7391897439956665, 0.0765770748257637, -1.6001160144805908, -1.046441912651062, 0.5922218561172485, 1.0386450290679932, 0.35836783051490784, 0.14602892100811005, -0.6350112557411194, 0.36258938908576965, 0.11749482899904251, 0.07251396775245667, 0.10954247415065765, 0.025561079382896423, 0.09631967544555664, -0.15594036877155304, -0.5407061576843262, 0.8875080943107605, 0.8932006359100342, -0.2725006937980652, -1.032094120979309, 2.199944257736206, 1.073548674583435, 1.335497498512268, -0.3522357642650604, -1.391905426979065, 1.4018901586532593, -0.3840349316596985, 1.8272048234939575, 0.7310769557952881, 1.3132951259613037], [-0.3534640967845917, 0.5506928563117981, 1.1235454082489014, 0.7723747491836548, 1.49750816822052, 1.1193054914474487, 0.1317436546087265, -0.6732020378112793, -1.6070880889892578, 0.7312780618667603, -1.4233015775680542, -0.7225989103317261, 0.5552958250045776, 0.5117445588111877, -0.3178001046180725, 1.0921601057052612, 3.1501381397247314, -0.8879659175872803, 0.8458857536315918, 1.0506194829940796, 0.04099831357598305, -1.06455659866333, 1.6307098865509033, -1.6741479635238647, -0.8539771437644958, -1.2646079063415527, 0.4172605872154236, 0.0684901773929596, -0.03000972792506218, -1.172243356704712, 0.9902759194374084, -1.5888147354125977, -1.275473952293396, -1.4930980205535889, -0.6444542407989502, 0.9858022928237915, -1.4077881574630737, -0.8724355697631836, -0.23416642844676971, 0.6185197830200195, 0.7345200777053833, -0.7486121654510498, 0.9962650537490845, 1.015366792678833, 0.4089900255203247, 0.1617588996887207, 1.2650278806686401, 0.11504939198493958, 1.7174044847488403, -0.38754037022590637, -0.9067336320877075, 1.1798747777938843, -1.3737565279006958, 1.5725421905517578, 0.5992121696472168, -1.090794324874878, -0.7959438562393188, -0.5422112345695496, -2.218442678451538, 0.21256008744239807, -2.1320667266845703, 0.8741200566291809, -1.2423442602157593, 3.269934892654419, 0.7184888124465942, -0.6689645051956177, -0.7874033451080322, 0.30443161725997925, -1.8188538551330566, -1.5010788440704346, 0.41210508346557617, -0.2294624149799347, 1.917935848236084, 0.08057639747858047, 1.3163319826126099, -0.4080986976623535, -0.3874613046646118, 1.0392200946807861, -0.30932286381721497, -2.3180744647979736, 0.828747570514679, -0.3657194972038269, -0.08617417514324188, 0.585940957069397, -0.7243250012397766, -0.7303438782691956, 1.989211916923523, 0.1638580858707428, 1.018198013305664, 0.14816179871559143, -0.5769339799880981, -1.3977471590042114, 0.4421417713165283, -0.3597002625465393, -1.4861196279525757, -0.8630276322364807, 1.189535140991211, -0.038762036710977554, -1.054280400276184, -0.04032985493540764, 0.4856022894382477, -0.4196805953979492, 0.9819391965866089, 2.0133204460144043, -0.667819082736969, -0.41872715950012207, -0.09216660261154175, -0.8794824481010437, -0.5161119103431702, 0.06579501926898956, -0.3247211277484894, -0.3245902359485626, 0.9007464647293091, -0.42293524742126465, -0.38609597086906433, -0.8948697447776794, -0.20168964564800262, 0.2765231430530548, -0.3241818845272064, -0.875701904296875, -0.7813073992729187, 0.8005658984184265, -0.37311041355133057, 0.2561577558517456, -1.2017031908035278, -0.971924364566803, 1.2558097839355469, -1.0818555355072021], [-0.6804041266441345, 0.32543137669563293, -0.8097848296165466, -0.2625347077846527, -0.11196092516183853, -0.010372907854616642, -0.9308649897575378, -0.5845635533332825, 0.7393662929534912, -0.28516143560409546, 1.6202408075332642, 1.8508522510528564, 0.9308972954750061, -0.03198263421654701, 0.6467229127883911, 0.19704805314540863, -1.6728322505950928, -1.1924382448196411, 0.5965171456336975, -0.5088545680046082, 1.030879020690918, 0.48393556475639343, 1.090716004371643, 0.07221375405788422, -0.7821584939956665, -0.6323122978210449, 0.9181134700775146, -0.5220168232917786, 0.6924683451652527, 0.17223255336284637, 0.2607109844684601, 0.1665390133857727, -0.8422961235046387, 0.20011650025844574, 1.1752954721450806, -0.3564068078994751, -0.06615413725376129, -0.38353899121284485, -1.5351626873016357, 0.33776339888572693, -1.396956205368042, 0.07204543799161911, 2.001378059387207, 1.3377587795257568, -1.2690250873565674, -1.3764876127243042, -2.523134469985962, 1.5678527355194092, 0.022357482463121414, 0.5982815623283386, -2.543987274169922, 0.7847080230712891, -0.44085219502449036, -0.5021742582321167, 0.8161281943321228, 1.1990259885787964, 2.067660093307495, -0.7969453930854797, -0.537634551525116, -0.5783528089523315, -0.8449276089668274, 2.2013134956359863, -1.5675312280654907, -0.18873783946037292, 1.2435065507888794, 0.37158674001693726, -2.107945680618286, 1.3472223281860352, -0.034413255751132965, 0.21857289969921112, -0.15371982753276825, 2.550153970718384, 0.195419579744339, -0.21005231142044067, 0.7421029806137085, 0.008736246265470982, 0.29887154698371887, -0.32347166538238525, -1.4655094146728516, 0.08949923515319824, -2.0299315452575684, -0.6478903889656067, 0.8664385676383972, 0.4156845808029175, -0.2705937922000885, 0.1490640640258789, 0.0345722995698452, 0.6305489540100098, -0.051897987723350525, 0.6886022090911865, -0.01762746460735798, 0.3563857674598694, 1.7360427379608154, 0.12435146421194077, 0.8894402384757996, -0.3595265746116638, -0.3869439363479614, -1.3643332719802856, 0.7445081472396851, -0.8464866876602173, 0.2103521078824997, 1.42634117603302, 0.3599783182144165, -0.3803669512271881, -2.757962703704834, 0.49653613567352295, 0.40886232256889343, -0.37911951541900635, 1.6868515014648438, 0.20675596594810486, 1.219359278678894, 0.9187009334564209, 1.0834165811538696, 0.20812396705150604, -1.3644814491271973, -0.21914665400981903, -1.3684414625167847, 0.8825092911720276, 0.4284112751483917, 1.4069007635116577, 0.4152478575706482, 0.9993048310279846, -1.1435929536819458, 0.6340488791465759, -0.23874402046203613, 0.258809357881546, -0.7301089763641357, 0.6146978735923767], [0.6370798349380493, -1.8175313472747803, 0.4866507947444916, -0.4459938108921051, -1.1918758153915405, -0.931973397731781, 0.0882231816649437, -0.41277772188186646, -1.843024492263794, -0.9924567341804504, 0.07062900811433792, 0.2247147411108017, 0.35762178897857666, -1.909929633140564, 0.7326352596282959, -0.6059151887893677, 0.3342778980731964, -0.14920099079608917, 0.6588447093963623, 2.4969496726989746, 1.5419749021530151, 0.6110708713531494, 0.5731682777404785, -0.7112318277359009, 0.5133885145187378, 0.3296060562133789, 0.07241465896368027, 0.8787316083908081, -0.9162952899932861, -0.05688738450407982, -1.0693997144699097, 0.5421831607818604, 0.8597774505615234, -0.7195088267326355, 1.2703003883361816, -1.0121532678604126, 0.6198654174804688, -0.18986566364765167, -2.0422558784484863, -0.9500692486763, -0.6930499076843262, -0.41666844487190247, -0.38937029242515564, -0.6950388550758362, -0.024379340931773186, 0.341705322265625, -0.05128377303481102, 1.3440028429031372, -1.1359716653823853, -0.6781301498413086, -0.3203122019767761, 0.22897261381149292, 0.5910891890525818, -2.8280599117279053, -0.9190247058868408, -1.0759814977645874, 0.2322973906993866, 0.9935697913169861, 1.597041368484497, -0.057927269488573074, 0.299381822347641, 0.16832590103149414, -0.7263746857643127, -0.6322574019432068, -1.3056849241256714, -0.2046869397163391, 0.4919046461582184, 0.305149108171463, 1.0636324882507324, -0.4677942395210266, -1.4681183099746704, -0.6432753801345825, 1.5414315462112427, 0.8157193660736084, 0.23543347418308258, -0.28471216559410095, 0.9748085141181946, -0.7578650116920471, 0.7541871666908264, 0.12106936424970627, 1.5163905620574951, 0.5406962037086487, 0.1412438601255417, 0.051450494676828384, -0.7637428045272827, -0.040543634444475174, 0.7013237476348877, 3.0129942893981934, 0.16165365278720856, 0.8037144541740417, 0.06780027598142624, -0.4991644024848938, 0.7696542143821716, 0.5430051684379578, -0.31404370069503784, -0.742429792881012, -1.6938258409500122, -0.007797385565936565, 1.9733856916427612, 0.47398263216018677, -0.24551647901535034, 1.2206404209136963, 0.17014464735984802, -0.9091746807098389, 0.37733393907546997, -0.17730778455734253, -2.4966773986816406, 0.9482818245887756, 0.4135868549346924, 0.655765950679779, -2.5437169075012207, 2.0704336166381836, 1.7166374921798706, 0.4579642117023468, 0.08620196580886841, 0.7645164728164673, 1.0930081605911255, 3.0603394508361816, -0.031854838132858276, 0.03343501314520836, 0.46627694368362427, -1.8721681833267212, -0.33309459686279297, 0.175310879945755, -0.0003743361448869109, -1.4572694301605225, -1.9298834800720215, 1.3666001558303833], [0.7437989711761475, -0.11354796588420868, 0.0419461727142334, 0.45738092064857483, -0.8771273493766785, -0.3529185950756073, -0.20646575093269348, 0.5325681567192078, 0.6618662476539612, 0.2253981977701187, -0.9854363799095154, 0.028728915378451347, 2.392008066177368, -0.009962466545403004, 0.4481569230556488, -0.39876729249954224, 1.8127005100250244, -0.1630454808473587, -2.5617151260375977, -0.4438408315181732, 0.9628298878669739, -1.0855777263641357, -0.27414408326148987, -0.10576590150594711, -0.09653809666633606, -1.2536990642547607, 1.411331295967102, 0.08712618052959442, 0.9314472079277039, 2.5981454849243164, 0.04560735449194908, 0.7655969262123108, 0.9933832883834839, 1.6218262910842896, 1.4069499969482422, 0.013059000484645367, -0.4103282392024994, -0.6225892901420593, -0.2936415672302246, -0.6561352610588074, 0.23286974430084229, 1.6199243068695068, 0.008239404298365116, 0.9035786390304565, 0.8447625041007996, -1.1329914331436157, 2.726304054260254, -0.6612989902496338, 0.009477206505835056, -0.4258006513118744, 0.0838417336344719, 0.7376950979232788, -1.9967632293701172, 1.1196690797805786, 0.1294933557510376, -0.48285815119743347, -1.2667306661605835, -0.7519830465316772, -0.8827942609786987, -0.11870308220386505, -1.3207824230194092, 1.6303203105926514, -1.0404466390609741, -1.6208592653274536, -0.48972561955451965, 0.6462391018867493, -0.3989124298095703, 0.3885112404823303, -2.7828776836395264, -1.5476703643798828, 0.14597147703170776, -0.12547902762889862, -0.1985716074705124, -1.0518156290054321, -0.1378738433122635, -0.6795271635055542, -1.0963051319122314, 1.6076483726501465, 0.31564071774482727, 1.3624651432037354, 0.12210449576377869, -0.7130804061889648, 0.9804190397262573, 1.0243630409240723, 0.26428651809692383, -1.0064308643341064, 1.3100061416625977, 0.7437922358512878, 0.5761165618896484, 0.5111591815948486, 0.3560100793838501, 1.7211921215057373, -0.32408371567726135, -0.7470819354057312, 0.7847375869750977, 0.7146501541137695, 3.4964256286621094, 0.8392844200134277, 1.2358161211013794, 1.7715959548950195, -0.15206769108772278, 0.19129255414009094, 1.2131093740463257, 1.0929614305496216, 0.7151090502738953, -0.3510848581790924, 0.586783230304718, -2.1884827613830566, -0.7684250473976135, 1.2011157274246216, -2.2720987796783447, -0.8486828207969666, 0.32294437289237976, -0.5454268455505371, 0.2567129135131836, -0.3603636920452118, -0.3754519522190094, -1.8147602081298828, -0.3700697720050812, -0.11214817315340042, -0.810030460357666, -0.962097704410553, 0.8730619549751282, 0.07202287018299103, -0.8983675241470337, -0.16203507781028748, -0.5217795372009277, -0.7404146194458008], [-0.2973375618457794, -0.03520958125591278, -0.25169673562049866, -2.226073980331421, -0.46091949939727783, -0.03855069354176521, -0.37723687291145325, 1.0587972402572632, -0.17032790184020996, -0.6458597183227539, 0.5986124277114868, 0.4216858446598053, -0.262029230594635, 0.22805078327655792, -0.17575131356716156, -0.1624567061662674, -0.07993044704198837, -1.8091753721237183, 1.4067790508270264, 0.8706493377685547, -0.7598238587379456, -0.1848057508468628, 0.30662915110588074, 0.9875702857971191, 0.9899836778640747, 0.8325196504592896, -0.2065257430076599, -0.9134083986282349, 1.2892733812332153, 0.9904146194458008, -0.28139445185661316, -1.7334169149398804, 0.25898993015289307, -0.011488315649330616, -0.0036527153570204973, 1.0765190124511719, 0.6425626873970032, 0.941593587398529, 0.09791690856218338, -1.3047226667404175, 1.0308507680892944, 1.0098267793655396, -1.011330485343933, -0.8785686492919922, 2.0647335052490234, 1.7991771697998047, -0.6377159357070923, -0.779600977897644, -0.6112164855003357, 0.3498663306236267, -0.6983497738838196, 0.797257125377655, 0.9049500823020935, 1.2908316850662231, 0.9838460087776184, -1.3020676374435425, 0.7701166868209839, 0.3174744248390198, 1.8337593078613281, -2.0981922149658203, -1.2717832326889038, 1.0764241218566895, 0.5796056985855103, 0.7568079233169556, 0.06612030416727066, 0.5160870552062988, -0.39720356464385986, 0.1355503499507904, 1.1206223964691162, 1.0060153007507324, -1.3322649002075195, 0.4230049252510071, 0.9158532023429871, 0.7442010045051575, 0.09458135068416595, 1.2459861040115356, 0.6094084978103638, 0.5657076835632324, 0.6666818857192993, -1.0327073335647583, 0.13863520324230194, -0.6747426986694336, 0.7459639310836792, -0.2979833781719208, 0.49033159017562866, 0.5708236694335938, -1.8335421085357666, -0.15970827639102936, -0.863289475440979, 1.628622055053711, -1.4727400541305542, 0.08143415302038193, 1.3412407636642456, -0.5341415405273438, -1.9739294052124023, -0.7104476690292358, 0.37574219703674316, -1.2930364608764648, -0.519273579120636, -0.4570220708847046, 2.126234769821167, -0.4720793664455414, 0.3250398635864258, -1.4588121175765991, 0.19195079803466797, 0.28009992837905884, 0.6534373164176941, -0.40708446502685547, 1.3748987913131714, 0.31221887469291687, 1.2764166593551636, 0.2432786077260971, 1.036885380744934, -0.537813663482666, 1.509490728378296, 0.20689094066619873, 0.661693274974823, -1.8539589643478394, -0.11862587183713913, 1.1845636367797852, -0.8768914341926575, 0.5390234589576721, 0.10566147416830063, 0.8065757155418396, 0.9534291625022888, -0.9318389296531677, 2.7964017391204834, -1.2194651365280151], [1.1972382068634033, 1.2890931367874146, 1.4600884914398193, 1.4476916790008545, 1.8165570497512817, -1.4541789293289185, 1.4547526836395264, 1.5829696655273438, 0.07956359535455704, 0.49322953820228577, -0.10586701333522797, 0.1800125688314438, -0.6522083282470703, -1.7719415426254272, -0.42458251118659973, -0.7219791412353516, 0.8604075312614441, -0.8763561844825745, -0.19982078671455383, 0.11107586324214935, -0.4448302388191223, 0.643783450126648, -0.05956358090043068, 1.341058611869812, 0.5735312104225159, -0.92320317029953, -0.9911594986915588, 0.41018664836883545, 0.028551287949085236, -1.0041338205337524, 0.8278215527534485, -2.659942865371704, -1.269814133644104, -1.2626694440841675, 0.7048355340957642, 1.0943849086761475, 0.17072328925132751, -0.66692715883255, -0.24847422540187836, -0.3470500409603119, -1.7448773384094238, -0.40076032280921936, 0.19969722628593445, 0.5535138845443726, -0.5358095765113831, 0.3013915419578552, -0.5723943710327148, -0.67413729429245, -0.5597115159034729, 0.625575840473175, -0.22335518896579742, -0.43173107504844666, -0.6867623925209045, 1.649782419204712, 1.1778022050857544, -0.8073751926422119, 0.008268672041594982, 0.5199518799781799, -0.5037801861763, 0.7015950083732605, 3.0762674808502197, -1.1479148864746094, 0.9050209522247314, -0.5555834174156189, -0.3985148072242737, 1.1662451028823853, -0.6441691517829895, -0.18717263638973236, -1.8960225582122803, -0.9784349799156189, -0.3150501549243927, -0.32310810685157776, 4.6749024477321655e-05, 1.5355875492095947, 2.0311121940612793, -1.2737668752670288, -0.3872087001800537, -0.1629868894815445, 0.27913087606430054, 0.509772002696991, -1.0635608434677124, 0.9022485017776489, 0.8615485429763794, -0.7230330109596252, 0.5403497815132141, 0.4419559836387634, -0.18041642010211945, -0.8467296957969666, 0.47420772910118103, 0.026039352640509605, 0.4003094434738159, -0.3386659324169159, 0.5586936473846436, 0.714949369430542, 0.17962470650672913, 1.473393440246582, -1.5235178470611572, -0.1384398490190506, 0.7739225625991821, 0.6298938393592834, 1.1335933208465576, -0.3477800786495209, -0.2418559193611145, 1.2988090515136719, -1.1018973588943481, 0.8590473532676697, 1.4544603824615479, 0.12161213904619217, -0.17034879326820374, 1.836063027381897, 0.6642548441886902, -0.48746493458747864, -0.7027187347412109, 0.4348240792751312, 0.011033877730369568, -0.862519383430481, -0.1955001950263977, 1.780346155166626, 0.8505296111106873, 1.7411160469055176, 0.27023807168006897, -0.46932709217071533, 0.5451105833053589, -1.6123073101043701, -0.42531970143318176, 0.5096415281295776, -1.373806118965149, 1.3286608457565308], [-0.4601983428001404, 1.7157186269760132, -1.3532977104187012, 0.6089658737182617, -0.29084616899490356, 0.6411770582199097, -0.15236423909664154, 0.12855572998523712, -1.7384709119796753, 0.16501088440418243, 0.4166755676269531, -0.5243573784828186, -0.38458701968193054, -0.32148826122283936, -0.20999230444431305, 1.4550927877426147, -0.3287866413593292, -0.46057385206222534, -0.7600634098052979, 0.4715062081813812, -0.4480604827404022, 0.5851181745529175, 1.8998874425888062, 1.1857274770736694, 0.30669838190078735, 0.6775880455970764, -0.035732343792915344, -1.6867743730545044, -0.1582910418510437, 0.5357309579849243, 0.06536245346069336, 0.6663979291915894, 0.20198389887809753, 0.5457658767700195, 0.8687990307807922, 0.7622513771057129, -0.47897276282310486, 0.05854896456003189, -0.17325223982334137, -1.714684009552002, 0.6459184885025024, 0.31468892097473145, 0.024837344884872437, -1.439274787902832, -0.28910428285598755, -2.0089194774627686, -0.15019771456718445, -1.1608716249465942, 1.6835986375808716, -0.0020766332745552063, 1.1450129747390747, 0.5990259647369385, 0.5507984757423401, -2.4003639221191406, 0.28478989005088806, -0.5277193188667297, -1.2960797548294067, -0.11256616562604904, -1.480505347251892, 0.8824741840362549, -0.20068225264549255, -1.5477385520935059, -0.8290485739707947, -1.0784376859664917, 0.03955969214439392, -0.9569100141525269, -0.14494365453720093, 0.5139186382293701, -0.11905259639024734, -0.6618125438690186, -2.211766481399536, -0.8602185249328613, -0.7449347376823425, -0.05111793056130409, 1.082645297050476, 0.41104739904403687, -0.2852638363838196, 0.7694353461265564, -2.21002197265625, 0.32769182324409485, -2.3358051776885986, -0.8418422341346741, -0.6546998023986816, -0.2960958778858185, 0.9977360367774963, 1.2056033611297607, -0.1259913593530655, 1.2595431804656982, 0.22420762479305267, 1.1988083124160767, -0.25338035821914673, -0.03225493058562279, -1.1751959323883057, 0.3201199471950531, 0.8747539520263672, -0.6656374931335449, 1.1283890008926392, 0.5354143977165222, 0.02175885997712612, 0.36391162872314453, -0.9363462328910828, 0.16021206974983215, -1.3691115379333496, -0.02601691521704197, -0.8402562737464905, 0.3292466998100281, -0.401755154132843, 0.19514240324497223, 0.3794945776462555, -1.1823015213012695, 0.5515711307525635, -0.9475882053375244, -0.025660617277026176, -0.4755396544933319, -1.4348434209823608, -2.507049798965454, 0.7734730839729309, -0.05070880800485611, -0.9522860050201416, -0.7403756976127625, 0.930871844291687, 1.3613991737365723, 1.20717453956604, 0.5031662583351135, -0.31059736013412476, 0.79571133852005, 0.35082727670669556, -2.3697495460510254], [-1.509514331817627, -0.3689460754394531, 0.09042089432477951, -0.555568516254425, -0.2576932907104492, -0.8847155570983887, 2.9286916255950928, -0.4520455002784729, 0.3027117848396301, -0.26713865995407104, -0.20667394995689392, -0.7598147392272949, 1.8941055536270142, 1.9831753969192505, -0.08819147199392319, 0.35756373405456543, 1.3090553283691406, -1.4050770998001099, 1.0417227745056152, 3.0171241760253906, -2.1290154457092285, -0.29754650592803955, 0.0737004354596138, 0.7057107090950012, -0.39648008346557617, -0.4584657549858093, -0.34565243124961853, 0.23537614941596985, -0.49960950016975403, -0.6163718700408936, 0.8890163898468018, -0.7320231199264526, 0.5352514386177063, 1.0821459293365479, 1.3708759546279907, -0.4809257388114929, 0.15147916972637177, 0.5356667041778564, -0.04423738270998001, 1.5535062551498413, -0.6260584592819214, 1.9625599384307861, -0.5701040625572205, -1.2522425651550293, 1.54082190990448, 0.8367566466331482, 0.4642186462879181, -0.812471330165863, -1.3354471921920776, -0.4175850450992584, -0.7794025540351868, 2.0141444206237793, -0.6150277256965637, -0.5828329920768738, 0.42438527941703796, -0.29900670051574707, 1.4327083826065063, 0.6560462117195129, 0.6537211537361145, -0.019052837044000626, -0.8632040619850159, -0.489680677652359, -1.3620020151138306, -0.20899352431297302, 0.8207676410675049, 1.721649408340454, -1.2266221046447754, 0.7253512144088745, 0.2661908268928528, 1.2153126001358032, -1.3771545886993408, -0.2272852063179016, -0.16784781217575073, 0.5984860062599182, -1.060454249382019, -0.5612750053405762, -0.4852268695831299, -1.2575761079788208, 0.6012894511222839, 1.4424694776535034, -0.12845751643180847, -2.0034468173980713, -2.3753855228424072, -0.4509451687335968, -0.37471604347229004, -2.0395779609680176, 0.9667499661445618, -0.12095074355602264, -0.8343600630760193, 1.2495551109313965, 0.041892699897289276, -0.9806046485900879, 0.186080202460289, 0.009042922407388687, -1.8575905561447144, 0.42875808477401733, -0.15565375983715057, 0.4866628348827362, 0.574647843837738, -1.1049152612686157, 0.9519853591918945, -1.1964248418807983, 1.5875937938690186, 0.6277761459350586, -1.3258345127105713, 0.6033731698989868, 0.35229718685150146, 1.311478853225708, -0.975641131401062, 0.6524600982666016, 0.34381014108657837, 0.7070921659469604, 1.5109519958496094, -0.02721489407122135, -0.051606230437755585, 0.43960797786712646, 0.19175903499126434, -0.7600454688072205, 0.8472272753715515, -0.4264480173587799, 0.6574605107307434, -1.0874749422073364, -1.71307373046875, -3.822582721710205, -0.8466974496841431, 0.18764185905456543, -0.1621733158826828, -0.22059696912765503], [-0.031224366277456284, -0.37512239813804626, 0.2147614061832428, 0.3918687403202057, -2.0718884468078613, -0.131989985704422, -0.3569897711277008, 0.6767802834510803, -1.2599647045135498, -0.6903677582740784, 0.07260338962078094, -0.7590836882591248, -0.2923429012298584, -0.4418477416038513, 0.1371639370918274, 0.923584520816803, 1.690759539604187, -0.7302415370941162, 0.28174149990081787, 1.402806282043457, 1.0206514596939087, -0.8412641882896423, 1.1691207885742188, 0.7467790842056274, -0.21547052264213562, -0.9501984119415283, -0.3401239812374115, -1.104145884513855, 0.9449641108512878, -0.22734591364860535, 0.41280290484428406, 0.8684067130088806, -0.4291398525238037, 0.626957893371582, -0.2044883370399475, -0.7345702648162842, -0.19247041642665863, 0.9184908270835876, 0.8078892827033997, -1.4662624597549438, 0.7124481797218323, -1.7534233331680298, -0.536489725112915, 0.5487954616546631, -0.6347485184669495, -1.9488621950149536, 0.7012184262275696, 1.2927955389022827, -0.334859162569046, -0.18943895399570465, 1.363416314125061, -0.2675037384033203, -0.6786037683486938, -0.23262692987918854, 0.5666512846946716, 2.0177712440490723, -0.1938718855381012, -0.03372958302497864, 0.9269269704818726, -2.1450650691986084, -1.4221441745758057, -1.445163607597351, -1.8013813495635986, 0.9290367364883423, -0.44324758648872375, -0.8563188314437866, 1.0484317541122437, 1.3440825939178467, 0.6099178791046143, 0.42929646372795105, 0.008121746592223644, -1.773158073425293, -1.8516823053359985, 1.4949692487716675, 1.2555629014968872, 0.10617661476135254, 1.4615790843963623, 0.7503129839897156, -0.7047680020332336, -0.528876781463623, -0.24762719869613647, 2.0736281871795654, -0.4652829170227051, -1.379085898399353, 0.4813919961452484, -0.12950672209262848, 1.327870488166809, -0.5761699080467224, 0.9930869340896606, -0.42351001501083374, -0.0641326904296875, 0.024122120812535286, -0.5828508734703064, 0.5602450966835022, -0.4576125144958496, -2.1217234134674072, 0.6758357882499695, 0.0857100561261177, 0.7473993301391602, -1.7055047750473022, -0.33519428968429565, -1.118340015411377, -0.41327103972435, -1.9153969287872314, 0.4991757571697235, 0.4030824601650238, -2.2603108882904053, 0.8762795329093933, -1.867984414100647, -0.8017050623893738, 1.3633116483688354, -0.29809045791625977, 0.5634652376174927, -0.59776371717453, -2.9104163646698, -0.26575347781181335, -0.5787888169288635, 1.0910134315490723, 1.8653596639633179, 0.2005317509174347, -0.23143444955348969, 0.0426105335354805, -0.6682046055793762, -0.40353503823280334, 0.899351179599762, 1.6598745584487915, -0.9025594592094421, -1.4872374534606934], [0.525827944278717, -0.9507789015769958, 0.8642174601554871, -0.08060600608587265, -0.6877934336662292, -0.33291521668434143, 0.3022500276565552, 1.5593560934066772, -0.9748132228851318, 0.2680862247943878, -0.561438798904419, -0.26188966631889343, 0.6134961247444153, -0.07123435288667679, -1.157068133354187, 0.4851830005645752, -0.19474251568317413, 1.0694944858551025, 1.1699258089065552, 1.5046881437301636, 0.03350881114602089, -0.3241189420223236, -0.6481562256813049, 0.8815291523933411, 0.7911455035209656, 0.6040866374969482, 1.3136539459228516, 0.8492633700370789, 0.2528519332408905, 0.45811790227890015, 0.8949987292289734, 0.002756438683718443, -1.0661405324935913, 0.01403167936950922, -1.2960211038589478, 2.4983859062194824, -0.7000892162322998, -0.07042161375284195, 0.6775902509689331, -1.1675208806991577, -0.7037816643714905, 2.9634666442871094, -0.48503121733665466, -1.049306869506836, 0.3298165798187256, 0.7584425806999207, 0.17096960544586182, -0.18505828082561493, 0.6706385016441345, 1.0235685110092163, -0.39029937982559204, 0.8549193739891052, 1.0046533346176147, -0.08707369118928909, -0.9341357350349426, -1.7298941612243652, 0.1700647473335266, 1.3064165115356445, 1.4078285694122314, 0.9677630066871643, -0.7378663420677185, -1.0939875841140747, -0.5764639377593994, -0.5374088287353516, -0.09007348865270615, 0.37728723883628845, -0.8028473258018494, -0.3217352330684662, 0.023508034646511078, -0.25679004192352295, 1.3141449689865112, 2.0478482246398926, 0.16387486457824707, 1.3294172286987305, -0.43853262066841125, 1.2618190050125122, 0.5368868708610535, -0.9902260303497314, 0.3177661895751953, -1.1885998249053955, 0.534489095211029, -1.8762072324752808, 0.017634300515055656, -3.310772657394409, -0.3976391851902008, -0.6800106763839722, 0.5059414505958557, -0.14612977206707, -1.3112032413482666, -0.9789907336235046, -0.0902879387140274, -0.2772059440612793, -0.3157809376716614, 0.3716847002506256, -1.1778032779693604, -1.2756050825119019, 0.18373413383960724, -0.6131279468536377, 0.08814098685979843, 1.023133397102356, 0.18962466716766357, 1.5484727621078491, -0.6530538201332092, 0.4596259891986847, 0.7269070744514465, -1.7588850259780884, -0.9338526725769043, -0.5058485269546509, 1.525684118270874, 1.0695053339004517, 2.0898165702819824, -1.5814406871795654, -0.132808655500412, 0.799256443977356, 0.49112293124198914, 0.23373554646968842, -0.8586317896842957, -0.5852786302566528, -1.7061495780944824, -0.2197338044643402, 0.08561538904905319, 0.7933294177055359, 1.655752420425415, 1.0268710851669312, 0.09883073717355728, 0.7001280784606934, -0.4386458396911621, 0.16569091379642487], [-1.074610948562622, -0.9978079795837402, -1.3206093311309814, -0.8641355633735657, 1.001049518585205, 0.7704272270202637, 0.09327897429466248, -0.45986419916152954, 0.6059487462043762, -0.7633399963378906, 1.2996056079864502, 1.5549129247665405, 0.8309623003005981, 0.20915082097053528, -1.6194493770599365, -0.23057471215724945, -1.6649060249328613, 1.1538569927215576, -0.27451565861701965, -0.4557006061077118, 0.8166406154632568, 2.438230514526367, 1.1650104522705078, -1.4461671113967896, 1.626636028289795, -0.5063916444778442, -0.5554666519165039, -1.992200255393982, -0.8937212228775024, 1.5001732110977173, -0.508720874786377, -1.98220694065094, -0.8021009564399719, 0.33387190103530884, -1.2575675249099731, -0.14445415139198303, 0.5634322762489319, -0.05922301113605499, 0.8144862055778503, 0.1308658868074417, -0.857760488986969, 1.791702389717102, -0.17210611701011658, -0.26895636320114136, -2.844498872756958, 0.9189245700836182, -0.40728071331977844, -0.6480641961097717, -0.6542546153068542, -1.0218706130981445, 1.889186143875122, 0.8721096515655518, 1.648963212966919, 1.3077311515808105, 1.4922056198120117, -0.18261784315109253, -1.4627829790115356, 0.4510452151298523, -0.6508822441101074, -1.1147805452346802, -0.4136531949043274, -1.8461743593215942, -1.66277015209198, 1.1480095386505127, -0.2572418749332428, 0.38334301114082336, 1.465349793434143, 1.9079461097717285, 0.19673879444599152, 0.40322020649909973, 0.0012075034901499748, 0.5248679518699646, -0.8677366971969604, -0.6855734586715698, -0.7158117890357971, -2.4921486377716064, 1.882541537284851, -1.2163059711456299, 0.2540597915649414, 0.5333380699157715, -0.7087024450302124, 0.2987653911113739, 0.812500536441803, -2.016685962677002, 0.9668771028518677, -1.1936367750167847, 1.842031717300415, -1.4482086896896362, 1.0925956964492798, -0.33817416429519653, 2.6372010707855225, 0.1448451429605484, 0.12772813439369202, 0.39485591650009155, 2.3052337169647217, -1.9929330348968506, -1.0233465433120728, -0.06347683072090149, 0.37859734892845154, -1.7362912893295288, 0.5670220851898193, 0.9039829969406128, 0.8014806509017944, 1.322049856185913, -1.1238648891448975, 0.3447321355342865, -0.47561734914779663, 0.06237802654504776, 0.7355695962905884, -0.758520781993866, -0.8805093169212341, -0.18512417376041412, 0.1101025938987732, 2.1131467819213867, 1.357417345046997, -1.1169946193695068, -0.3651115596294403, 0.4528239965438843, -1.848804235458374, 0.46256023645401, -0.7646980881690979, -0.9337769746780396, 0.7487106919288635, 0.7683749794960022, -0.18950553238391876, 0.9495747089385986, -1.1180553436279297, 0.3288296163082123], [-0.9196618795394897, -1.7287213802337646, -0.1491638422012329, -0.1958039253950119, 0.27861446142196655, -1.0219906568527222, 0.9374403953552246, -0.4287013113498688, -0.4337928295135498, 1.6036782264709473, 0.30024996399879456, 1.6817442178726196, 0.2449328899383545, 0.004066560883074999, -0.3556769788265228, 0.9807415008544922, -0.21658870577812195, 1.4146877527236938, -0.7840853333473206, -0.1385093331336975, 1.1448135375976562, 0.9546535611152649, 1.1583974361419678, 1.8666164875030518, 1.8531360626220703, -1.020470142364502, 0.731042206287384, -0.503017783164978, 0.8146718144416809, 1.0023231506347656, 0.4136436879634857, -0.7410396337509155, -1.3891202211380005, -1.3703279495239258, -0.4890788495540619, -0.288115918636322, -0.3956213593482971, 0.18793758749961853, -0.5704978108406067, -0.3636654317378998, -0.9219467043876648, 0.10671275854110718, -0.9151618480682373, -0.09264186024665833, -0.9261941313743591, 0.954663872718811, 0.7782430052757263, -0.257406085729599, -0.10744938999414444, 0.5471811294555664, 1.041268229484558, -0.0537288598716259, 0.5637459754943848, 0.4010971486568451, 1.5558128356933594, 0.007368507795035839, 0.32971304655075073, -1.060341715812683, 0.7591286301612854, -0.6194765567779541, 0.5217900276184082, 0.15591707825660706, -0.12489040195941925, -1.3011802434921265, -0.32867908477783203, -1.052351474761963, -0.881009042263031, 1.376927137374878, 1.2850862741470337, 1.5023349523544312, 1.3475674390792847, -0.7273058295249939, 0.8405656218528748, -0.304973840713501, -0.4816051125526428, 0.048206839710474014, 0.6340292096138, 2.002686023712158, -0.8285728693008423, -0.7096712589263916, 0.7820954918861389, 0.36485838890075684, 0.8861941695213318, -0.36793115735054016, 0.5888804197311401, 1.094019889831543, -2.0718724727630615, 1.2630013227462769, 0.8012531995773315, 0.28747156262397766, 0.07987529784440994, 0.14624322950839996, 0.018709300085902214, -0.04032731056213379, 0.7156181931495667, 1.8096014261245728, 0.828241765499115, 0.5129683613777161, 0.27573731541633606, -0.5339649319648743, 1.2118430137634277, -1.032893180847168, -0.6097216606140137, -0.20792385935783386, 0.5052871108055115, 2.2203619480133057, -0.33119192719459534, 0.2112504094839096, -0.9961045980453491, 2.395111560821533, 0.8441649079322815, 0.47665417194366455, -0.9419996738433838, -1.045236587524414, 0.502233624458313, 1.8182721138000488, 1.8088488578796387, -0.1524251252412796, 1.0109614133834839, -0.7552810311317444, 0.3282370865345001, -1.7919989824295044, -0.3553769290447235, 0.0726817324757576, 0.41168564558029175, -1.0478708744049072, -0.16749611496925354, 0.10823900997638702], [0.4681227505207062, 0.2904437482357025, -0.13278181850910187, 0.3466569483280182, -0.13861620426177979, 0.24188901484012604, 1.6180315017700195, -0.8640843033790588, -1.2333197593688965, -0.03942276909947395, 1.5342576503753662, 0.7805175185203552, -0.34704774618148804, -0.33204251527786255, 0.42560479044914246, -1.572103500366211, -1.9439547061920166, 1.7058491706848145, 2.273820400238037, 1.311975121498108, -0.4866578280925751, -1.7050808668136597, 0.2879011332988739, -1.7631640434265137, -0.8129501342773438, -1.4724828004837036, 2.6720569133758545, 0.09943576157093048, 1.432713508605957, 0.9759116768836975, -1.1419676542282104, -0.7607651948928833, 1.031643033027649, 0.7369300127029419, -0.39199453592300415, 0.9124220609664917, -1.4245661497116089, -1.219705581665039, -0.5305808186531067, 1.1178487539291382, 1.0483657121658325, -0.11335201561450958, -0.7913246750831604, 1.0598584413528442, -0.21518439054489136, -0.4097931385040283, 0.34335556626319885, -0.7854981422424316, 1.3345820903778076, 0.538414478302002, 0.6383141875267029, 1.0624184608459473, 0.6169366240501404, 0.20924142003059387, 0.8117552399635315, -0.11306212097406387, 1.7629905939102173, 0.24277201294898987, -0.9779278635978699, -1.6281206607818604, -0.2773127257823944, -1.6446037292480469, 0.3384285271167755, 1.2838109731674194, 0.9721166491508484, -1.543574571609497, 1.1634397506713867, -0.5766012668609619, -0.3229503035545349, 1.2721558809280396, -0.7957586646080017, -0.08542853593826294, -2.598947525024414, 0.4150645434856415, -0.6265376806259155, 0.24679520726203918, -0.9779201149940491, 1.475150227546692, 1.512573003768921, -1.3396459817886353, -0.8429805040359497, 0.41465359926223755, 0.623141884803772, -1.77153742313385, -0.6349992752075195, 0.8619882464408875, 0.6947504878044128, 0.8623543381690979, 0.5123451352119446, -0.6388125419616699, -0.45504602789878845, -0.5065708756446838, -0.06342645734548569, 1.2435001134872437, -0.5121187567710876, 1.3832166194915771, 0.3792792856693268, -1.5328044891357422, -0.5674404501914978, -0.010692579671740532, -0.4921151101589203, 0.24329215288162231, 1.5935437679290771, -0.3872714936733246, -1.1629348993301392, 0.24178552627563477, -0.2887722849845886, 0.8426342606544495, 0.05994049832224846, 1.7205615043640137, 0.37369975447654724, 0.5593997240066528, 1.4403884410858154, -0.6147626042366028, -0.08834714442491531, 1.4407093524932861, -0.8649987578392029, 0.4136406183242798, -1.5861616134643555, -1.6163251399993896, 0.44508299231529236, 0.700311541557312, 0.41052019596099854, -0.17570242285728455, 0.7127659916877747, 0.41152071952819824, -0.7601349949836731, 1.1913774013519287], [-0.6620993614196777, 1.6103888750076294, -0.6047072410583496, -0.982129693031311, 0.5691308975219727, -0.5986242294311523, -1.3268303871154785, -1.526434063911438, 0.5388636589050293, -1.9914478063583374, -0.7096996903419495, 1.4230061769485474, 0.06477317959070206, 1.9746880531311035, 1.6030887365341187, -0.6559116244316101, 0.9333539605140686, 0.6178748607635498, -2.1944119930267334, 0.03474392741918564, -0.19589467346668243, -1.4246292114257812, -0.7958099842071533, -0.9129156470298767, 0.3634471297264099, 0.5866894125938416, 0.6135447025299072, -1.148488998413086, -0.44474753737449646, -0.2011537402868271, -1.423324704170227, -0.5091969966888428, 0.47951817512512207, 1.4947497844696045, -1.1800709962844849, -0.06705302000045776, -0.2552317678928375, 0.5390483140945435, 0.42507249116897583, -1.390065312385559, -0.28968632221221924, 1.4958029985427856, 0.060761600732803345, 0.3650103509426117, 0.2752164602279663, -0.846846878528595, 0.6417477130889893, -0.9610976576805115, 1.2505298852920532, -1.2104997634887695, -0.776003360748291, -1.420397162437439, -1.5788599252700806, -0.5338925123214722, 0.22035925090312958, -1.1833621263504028, -0.7222675681114197, -0.3152472972869873, 1.0523808002471924, 0.01184858102351427, 0.9955674409866333, 0.6699885725975037, -0.15215526521205902, 0.8906643986701965, -1.3143651485443115, -0.6168785095214844, -0.0970817282795906, -0.17643727362155914, 0.4860825538635254, 0.38983967900276184, 0.12642721831798553, -0.36702871322631836, -0.8112855553627014, -0.5379928946495056, -0.596889853477478, -0.7681429386138916, -0.9594197869300842, -0.8772031664848328, -0.32761937379837036, 1.3977551460266113, -0.47018200159072876, -1.2260282039642334, -1.629944920539856, -0.9728997349739075, 2.3235080242156982, -0.40538179874420166, 0.7578083872795105, 0.06401252001523972, -0.635835587978363, 1.9792717695236206, -0.1326088160276413, 0.37652042508125305, 0.30122193694114685, -0.5177728533744812, -0.11043816804885864, -0.22665677964687347, 1.1464370489120483, -0.47183674573898315, -0.21708232164382935, 1.0109691619873047, -1.3298754692077637, -1.3502590656280518, -1.6063168048858643, -0.8886772394180298, 1.3857413530349731, -0.32797789573669434, 1.61770498752594, 0.08963152766227722, -0.25013765692710876, 0.48181042075157166, -1.2997053861618042, -0.583317220211029, 1.0135581493377686, 2.754211664199829, -0.36381959915161133, 1.9253484010696411, -0.5532252192497253, -0.19146883487701416, 0.024979671463370323, -0.3073577284812927, 1.7256793975830078, -0.034386783838272095, -0.0801912322640419, -0.41181039810180664, 0.9473317265510559, -2.330355167388916, -0.8681373000144958, -0.9728652238845825], [-0.3259774446487427, -0.28811123967170715, 2.2031054496765137, 0.6788741946220398, 0.47339603304862976, -0.6138579845428467, -1.5865973234176636, -0.8427490592002869, -1.082678198814392, 0.6705098748207092, 0.3660544455051422, 0.34398314356803894, -0.20865298807621002, 0.7715412974357605, 1.2476142644882202, -1.4019784927368164, 0.04082260653376579, -0.2818327248096466, -0.4530780613422394, 1.1814024448394775, 0.4092947542667389, 0.28805187344551086, -2.042891502380371, 0.38309037685394287, -0.2922815978527069, 1.5429794788360596, -0.0036659075412899256, -1.99934720993042, 0.5076224207878113, -1.0149173736572266, 2.0583581924438477, 0.36444127559661865, 0.6571493148803711, 1.0865509510040283, 1.2084977626800537, -0.14204348623752594, 0.44876715540885925, 0.9764418005943298, 1.1392929553985596, -0.0430060513317585, -0.4109119772911072, -0.4465409815311432, -0.9946406483650208, 0.6187107563018799, 0.016058219596743584, -0.09994454681873322, 0.07773828506469727, -1.4630142450332642, -0.3577705919742584, 0.15757757425308228, -0.7940547466278076, 0.04194119572639465, 0.14532381296157837, -0.18540741503238678, 0.17250218987464905, -2.20696759223938, 2.7030508518218994, -0.21454760432243347, -0.13624316453933716, 0.37690603733062744, 1.8549964427947998, -0.014904802665114403, -1.0481746196746826, -0.7470545172691345, 0.8225038647651672, 0.980626106262207, -0.2660464644432068, 0.19236716628074646, -1.1877528429031372, -1.2739449739456177, -1.0345749855041504, -0.6526355743408203, 2.146369218826294, -0.3056785464286804, -1.8505022525787354, -1.076988935470581, -0.39872753620147705, 0.7503153681755066, 0.21367645263671875, -0.26394814252853394, -0.8756751418113708, -1.1122108697891235, 1.651431679725647, -0.36723077297210693, -0.16030378639698029, -1.579978346824646, 1.7151850461959839, 1.2410987615585327, 0.8149164915084839, -0.19057703018188477, -0.11238245666027069, 0.5426734685897827, -1.153341293334961, -2.3246233463287354, -0.562593936920166, 1.3406352996826172, 0.34661275148391724, 0.7484866976737976, -0.906455397605896, -0.4224216043949127, 1.11231529712677, -0.1316179484128952, 0.6447846293449402, 0.42733466625213623, -0.18914379179477692, -0.6912724375724792, 1.0425071716308594, 0.03743595629930496, -2.399817705154419, -0.48975515365600586, -0.3477858901023865, 0.2104707658290863, 1.9453948736190796, 0.3605524003505707, 1.2465516328811646, 0.30850499868392944, 0.5177515149116516, 0.5995132327079773, -0.08219338208436966, -0.8767297863960266, 0.39485886693000793, -0.40338432788848877, -0.39120855927467346, 0.5315198302268982, -0.7794243693351746, 1.8042031526565552, 2.691391944885254, 0.22257056832313538], [1.9926981925964355, 1.4058023691177368, 2.968702554702759, -1.4811760187149048, -0.06027279421687126, -1.3678882122039795, 3.616220474243164, 1.4313658475875854, 0.15427491068840027, -1.907597303390503, 1.3508093357086182, 1.001157522201538, 0.4505631625652313, 1.177535891532898, 0.42879584431648254, 0.29957690834999084, 0.919662356376648, 0.01993997022509575, 0.17396418750286102, -2.066019296646118, 0.5814497470855713, 0.2966271936893463, 1.8086758852005005, -0.14828349649906158, -0.6262916326522827, -2.1030421257019043, 0.16642531752586365, -1.2335829734802246, -1.014328956604004, -1.2788199186325073, -1.8232566118240356, -0.4456404447555542, -1.0728346109390259, 0.4680439233779907, 0.4827134907245636, 0.4857160747051239, 0.35336822271347046, -0.062162235379219055, 0.9551058411598206, 0.9346866011619568, 3.7664406299591064, -0.8304390907287598, 0.5132840871810913, 0.47617971897125244, 1.392352819442749, -0.796371579170227, -0.05101092532277107, 0.14665308594703674, 1.585288166999817, 0.9604313969612122, -0.6496290564537048, 1.7413921356201172, -0.13061672449111938, -0.2731309235095978, 0.9965574741363525, -0.6325552463531494, -2.1317851543426514, -1.3929054737091064, -0.27955740690231323, -1.2652298212051392, 0.13704094290733337, 0.7083151340484619, -0.2177516371011734, 0.49729058146476746, 0.9896579384803772, -1.2923721075057983, -1.3024585247039795, 2.16127610206604, -0.04363070800900459, 1.3408035039901733, -1.044305682182312, 1.2618550062179565, -1.1780234575271606, 1.1332281827926636, -0.34109005331993103, -0.7557936310768127, -1.6834830045700073, 1.5766823291778564, 0.49034151434898376, -0.13243837654590607, -0.566685140132904, -0.260957270860672, -1.3315551280975342, -1.1857563257217407, -0.7350142002105713, 1.273310899734497, -0.003393962047994137, -0.6582104563713074, 0.35264432430267334, 0.1578475385904312, 3.149354934692383, -0.2086050659418106, 0.17486850917339325, -0.683473527431488, 1.3494952917099, 1.5935237407684326, -0.8494670391082764, -0.08620300143957138, 0.13609814643859863, 0.5280500650405884, 1.322624683380127, -0.9547944068908691, -0.8138295412063599, 0.8672853112220764, 1.6469889879226685, -1.0930161476135254, 0.6716851592063904, -0.013794687576591969, -0.7431960105895996, 1.7984205484390259, 1.3724498748779297, -0.3032735586166382, -0.18131189048290253, -0.850201427936554, 0.21190601587295532, 0.6430598497390747, 0.6000176072120667, 1.5033154487609863, 0.03934710472822189, 1.0542104244232178, 1.0921164751052856, -0.7221499681472778, -0.567391574382782, 0.9480128288269043, -0.5392935276031494, 1.603635311126709, -0.9666173458099365, -0.09941236674785614], [0.279122531414032, 2.1076347827911377, -0.07671777158975601, -0.9488210678100586, 0.9750963449478149, 0.27739420533180237, 0.7543531656265259, 0.6021396517753601, 1.3608512878417969, 1.1035090684890747, 2.581472635269165, -1.2286460399627686, 0.5225597620010376, -0.04482525587081909, 0.3695809245109558, 1.3012921810150146, 0.7668343186378479, -0.9569141268730164, 0.2053537517786026, 0.7605693936347961, -0.1551106572151184, -1.0598238706588745, -0.07044205069541931, -1.4628150463104248, 0.5905495285987854, 0.5170887112617493, 0.42514243721961975, -0.5758149027824402, 0.43910497426986694, -0.42437979578971863, 1.1030739545822144, -0.506361722946167, 1.7157371044158936, -0.5065728425979614, -0.913571298122406, -0.6940438151359558, 0.4287305772304535, -0.6474999189376831, -0.9461334347724915, -0.422249972820282, 0.026684381067752838, 1.7370429039001465, 0.3047393262386322, -0.5513588190078735, 0.4119945466518402, 1.6268634796142578, 1.4531874656677246, 0.34019935131073, -0.8735553026199341, -1.6781790256500244, -0.46546506881713867, 0.2711125314235687, 2.1396942138671875, -0.5471006035804749, 0.9000056385993958, -1.2852803468704224, 1.69222891330719, 0.8097756505012512, 0.43997201323509216, -0.6527765989303589, -1.1013226509094238, -0.24175341427326202, -0.03710946440696716, 0.542445719242096, -0.9146426916122437, 1.087125539779663, -1.2096387147903442, 0.305081307888031, 0.5044463276863098, -1.391957402229309, -0.13964004814624786, 0.0020135617814958096, -0.012013910338282585, -1.9707043170928955, -0.41489264369010925, 0.6207966804504395, -0.198699951171875, 0.30700644850730896, 1.1008622646331787, 0.5889461636543274, 0.6966391205787659, -0.8406205177307129, 0.5592449307441711, 0.39859771728515625, -0.35758236050605774, 1.270513892173767, -1.1013548374176025, -0.4437709450721741, 0.8036463856697083, 1.0880186557769775, 1.5890612602233887, -0.6549800038337708, 1.9942991733551025, 1.7117533683776855, -1.4536230564117432, -0.848849356174469, 1.6876682043075562, 0.5341471433639526, -0.07249964028596878, -1.4317547082901, -0.3639697730541229, 0.15095800161361694, 1.5569363832473755, 1.851758360862732, -1.058203935623169, -0.3957469165325165, -0.12108700722455978, -0.5285400152206421, -0.19818046689033508, 0.6866239905357361, 1.509106159210205, -0.12607495486736298, 0.6683463454246521, 0.4194137752056122, 0.600058913230896, -0.15280792117118835, 0.4022162854671478, 0.37371566891670227, -0.8022723197937012, 0.364270955324173, -0.08747054636478424, 0.30075517296791077, 0.41486117243766785, -0.37706732749938965, -1.747657299041748, 0.02164984866976738, 0.16478845477104187, -0.8981328010559082], [-0.13903303444385529, 0.06602021306753159, -1.167346715927124, 1.3095393180847168, 2.0106523036956787, 0.8927978277206421, -0.19109377264976501, 0.39663127064704895, 1.4656026363372803, -0.671330988407135, -2.4707634449005127, -0.40543729066848755, 1.4977163076400757, -0.8059950470924377, -1.0873500108718872, -0.8591304421424866, -0.8747830986976624, -0.23063445091247559, -0.7773882746696472, -0.5443016886711121, -0.146551251411438, 2.0194859504699707, -2.5505387783050537, -0.9225045442581177, 0.23117195069789886, -0.36799320578575134, 1.1977488994598389, 1.353004813194275, 0.3333771824836731, 0.5199328660964966, -2.532224416732788, 0.3177785277366638, -1.3931853771209717, -0.1524633765220642, 1.3334800004959106, -0.24388086795806885, -0.5841124057769775, -0.29559826850891113, -0.15841808915138245, -0.7154936194419861, 0.4873853325843811, 0.9702150225639343, 0.6392775774002075, -0.9478472471237183, -0.8470709919929504, 0.2512061595916748, 0.6690705418586731, 0.5198219418525696, -0.24113868176937103, 0.6648495197296143, -1.4347413778305054, -1.0354487895965576, -0.8806383609771729, -1.25295889377594, 0.2989976108074188, 0.30948498845100403, 1.3168983459472656, -0.4540809690952301, -0.28120025992393494, -1.6095774173736572, 1.2485462427139282, -0.24485211074352264, -0.730504035949707, -0.3822292983531952, 0.6385628581047058, 1.416020154953003, 0.6071968078613281, -2.230743646621704, -0.13752521574497223, -0.8721665143966675, -1.4271905422210693, -0.11338590085506439, -0.49693191051483154, 0.2618204355239868, -0.28070712089538574, -0.08639122545719147, -1.1694903373718262, 1.8890388011932373, -0.0936380922794342, 1.0224218368530273, 0.6173871159553528, 0.2782335579395294, 0.8845202326774597, 0.5137075781822205, -0.6753848791122437, 1.3342365026474, 0.8279340267181396, 1.4663114547729492, -0.8800851702690125, 0.857954740524292, -1.1457475423812866, -1.122507929801941, -0.12161347270011902, 0.544864296913147, 1.4300564527511597, -2.7789061069488525, 0.08386284112930298, 0.452572226524353, -1.41896390914917, -0.1899852305650711, 1.7786685228347778, 1.3011400699615479, -0.20437222719192505, -0.2631521224975586, 0.012074715457856655, 0.3233412206172943, -0.7508811950683594, 0.38254907727241516, 0.8251510262489319, -0.08475053310394287, 0.09967731684446335, 0.5647013783454895, 0.6673690676689148, 1.8052797317504883, 1.1524219512939453, 0.10174059867858887, 2.125392436981201, 0.9156227111816406, 0.248674213886261, 3.058110475540161, 2.090238094329834, -0.0167061910033226, 0.7289265990257263, -0.3421977758407593, -0.8466348648071289, -1.1794087886810303, -1.4867477416992188, -1.0856791734695435], [-1.2383160591125488, 1.568437099456787, -1.3192787170410156, -1.5135948657989502, 0.5193935632705688, 0.3035900592803955, 1.247172474861145, 0.13191282749176025, -1.2467727661132812, 0.5969904661178589, 0.09616678208112717, -0.1547747552394867, -0.7873863577842712, 1.3108460903167725, -2.429718255996704, -0.5370407104492188, 0.19780078530311584, 2.5860204696655273, -0.191285640001297, -0.26232683658599854, 0.5391270518302917, -0.7028310298919678, 1.4375399351119995, -0.6709633469581604, 0.4270976185798645, -1.0170689821243286, 0.20626628398895264, -1.125869631767273, -0.3081973195075989, -0.6716747879981995, -0.9853718280792236, 1.1536118984222412, 0.8556072115898132, -0.9571676254272461, -0.17875994741916656, 0.37613821029663086, 1.2592099905014038, 0.48592689633369446, 0.33810490369796753, -0.7603920102119446, 0.5948973298072815, -0.1969200074672699, 0.8621121048927307, -0.05828092619776726, -0.29177534580230713, 0.8266951441764832, 1.3827952146530151, 0.9308211803436279, 0.1227603480219841, -0.9528909921646118, 0.7539242506027222, -0.5104163885116577, 0.3677355945110321, 0.37924838066101074, -0.5478004813194275, -0.5895314812660217, -0.019157597795128822, 0.44681277871131897, 0.5579231381416321, -0.03825082257390022, -0.7113467454910278, 1.8529186248779297, -0.7180847525596619, 1.1783427000045776, 0.5865811109542847, 0.2913132607936859, -0.32320666313171387, -0.8328267931938171, 1.8690707683563232, 0.7976839542388916, 0.7407416701316833, -0.09560508280992508, 1.158666729927063, 0.682809591293335, 0.04987826570868492, -1.0819861888885498, -0.3806068003177643, 0.9034691452980042, -1.2559723854064941, 0.3566667139530182, 3.0634379386901855, 1.0757437944412231, 0.08184616267681122, 0.9663999676704407, 0.03842156007885933, -0.9032833576202393, -1.014224886894226, 0.7059679627418518, -1.1819382905960083, 0.6973797678947449, 1.2462198734283447, 0.10445693880319595, 1.842391014099121, 0.02156577631831169, 0.9937222003936768, -1.4815384149551392, -0.7385396361351013, -0.8867174983024597, -1.316084384918213, 0.7425537705421448, 0.664377748966217, 0.3564436435699463, 1.9166325330734253, 0.5342329740524292, -0.8699139952659607, 2.3924460411071777, 0.29271188378334045, 1.42911958694458, 1.1980020999908447, 0.2637402415275574, -0.3118532598018646, 0.9875304102897644, 0.9390414953231812, -0.030752897262573242, 1.5950406789779663, -0.24744315445423126, -0.5852847695350647, -0.38883480429649353, -0.7241402268409729, 1.1279386281967163, 0.7150726318359375, 1.196340560913086, -1.9544572830200195, -0.22563832998275757, 0.4113830029964447, 0.4535437822341919, -1.2767330408096313, 0.4403323829174042], [0.9924085736274719, 0.3154467046260834, 0.2795262634754181, 1.6672519445419312, -0.7568599581718445, 0.38853511214256287, -1.1328907012939453, -0.47157368063926697, -0.28539586067199707, 0.04221147671341896, 0.6942878365516663, 0.12291520833969116, -0.05227401480078697, -0.9225214719772339, -0.36043688654899597, 0.1883227378129959, 1.1057771444320679, 2.006748914718628, 0.5827542543411255, 1.2962825298309326, 0.35929766297340393, 1.1205774545669556, -0.37390241026878357, 0.022029398009181023, -0.41571044921875, -1.0199617147445679, -0.27446961402893066, 0.2679520547389984, 1.2888314723968506, -1.2668859958648682, 1.8328455686569214, -0.2662321627140045, 1.3057708740234375, 0.1687747985124588, 0.4462631046772003, -0.6042040586471558, -0.1785356104373932, -0.9378975033760071, 0.31100940704345703, -0.7799066305160522, 2.0065114498138428, -2.362295627593994, 0.06845235824584961, 0.6096667051315308, -1.521628737449646, 0.5486116409301758, 1.8495746850967407, -0.599219560623169, 0.4348025918006897, -1.003923773765564, 0.17610685527324677, -0.19514229893684387, -1.106677532196045, 0.12953335046768188, 0.5628364086151123, 1.0251399278640747, -1.1204825639724731, -1.2902960777282715, -1.829228162765503, -0.8917286396026611, -1.2253928184509277, -1.1313575506210327, -1.101321816444397, -1.072791576385498, 0.9819594025611877, -0.11333275586366653, 0.6644616723060608, -0.2945912182331085, 0.06042197719216347, -0.1430051028728485, -1.7159903049468994, -1.6099873781204224, -0.8683305382728577, -1.0572775602340698, 0.5957267880439758, 0.7568184733390808, -2.2345340251922607, -1.6908445358276367, -0.07766905426979065, -0.6313027143478394, -0.8992658257484436, 0.9767758250236511, -0.44258689880371094, 0.173512265086174, 1.7677104473114014, 0.26660192012786865, -0.11923612654209137, 0.3556959331035614, 0.30802395939826965, 0.6683254241943359, 0.1415046751499176, -0.3375481367111206, -0.1889275759458542, 0.8462281823158264, 0.4184674918651581, -1.03959059715271, -0.6012322902679443, -0.8792601227760315, 0.6348747611045837, 1.3217474222183228, -2.1789865493774414, -2.0722994804382324, -1.6452622413635254, -1.9813672304153442, 0.5195600986480713, 0.5721747279167175, 0.6957330703735352, -0.23283027112483978, -1.652726411819458, -0.08281341940164566, -0.6861754655838013, -0.9563804864883423, 0.5428809523582458, 0.23791208863258362, -0.9383929371833801, 0.3744959533214569, 0.1748623549938202, -0.16700617969036102, 1.778151512145996, -0.7305232882499695, 0.9431300759315491, -0.7361191511154175, -0.07395871728658676, 0.6755782961845398, -0.34259912371635437, -1.8321483135223389, 0.1455700546503067, 1.148250937461853], [-1.9274816513061523, -0.17618273198604584, 0.41245391964912415, 0.5024548768997192, 0.2490478903055191, -0.06714358180761337, -0.003385168267413974, 0.42663508653640747, -1.9893804788589478, 0.5751867890357971, -1.3406730890274048, 0.487829327583313, 0.5408231616020203, -0.9512893557548523, -0.030998097732663155, -0.49144628643989563, -1.0807219743728638, -1.427169919013977, 1.866140604019165, 1.8983235359191895, 0.4514476954936981, -0.652694046497345, -0.3579866290092468, -0.6852094531059265, 0.6068771481513977, -2.7930262088775635, -0.8386655449867249, 2.4982309341430664, 1.526297926902771, 0.03813338652253151, 0.5603317618370056, -1.5458780527114868, -0.3930949866771698, -0.07905204594135284, -0.47085678577423096, -0.035802584141492844, 0.012736459262669086, 0.5258583426475525, 1.2090332508087158, -1.634726881980896, 0.3237203359603882, -0.8765354752540588, 1.7616022825241089, -0.17308764159679413, 0.5012949705123901, 0.5812623500823975, 0.8436272144317627, 0.4656875431537628, 1.3158529996871948, -0.8353280425071716, -0.20864446461200714, -0.3214658796787262, -1.4328210353851318, 2.392960786819458, 1.363814115524292, -1.0600273609161377, 0.5896530747413635, 1.1401766538619995, -0.8419948220252991, 0.03550206497311592, 1.7093740701675415, 1.2753188610076904, -0.5298274755477905, 0.13744623959064484, 0.45025157928466797, -0.30678504705429077, 0.38547712564468384, 0.010622942820191383, 0.13775242865085602, -0.37347322702407837, -0.8147854208946228, -0.9551987648010254, -0.01906852424144745, -1.769195318222046, 0.9205381870269775, -1.1072958707809448, 0.9510531425476074, -0.4523358941078186, 1.3932912349700928, -0.20792341232299805, -0.24325917661190033, -0.6063350439071655, 1.6544103622436523, -1.926267385482788, -1.1387563943862915, -1.5400571823120117, -1.3149596452713013, -0.3203709125518799, 0.26825571060180664, -1.4256559610366821, -2.0130584239959717, -0.24960584938526154, 0.5572265982627869, -0.5497238636016846, 1.076372504234314, -0.062026530504226685, -0.8777298331260681, 0.7416831851005554, -0.5402707457542419, -0.4009617865085602, 0.3978212773799896, -1.3694086074829102, 0.147314190864563, -1.1776752471923828, -0.5852366089820862, -0.8597279191017151, 0.5971112847328186, -1.561689019203186, -0.0014509857865050435, -0.39505115151405334, -0.647907555103302, 1.530669093132019, 0.13664263486862183, 0.46128442883491516, -2.1374502182006836, 0.37661346793174744, 0.3557491600513458, 0.17849235236644745, -0.08805417269468307, 0.0954999178647995, 1.3297874927520752, -0.6382498145103455, 0.729119062423706, 1.7760969400405884, 1.8603847026824951, -1.502375841140747, -0.930615246295929, 1.0113009214401245], [-0.7691732048988342, 1.7016081809997559, 1.5476452112197876, -0.328575074672699, 1.6237610578536987, -2.9123945236206055, -2.4241225719451904, -0.9777380228042603, -0.36391836404800415, -0.8900195360183716, -0.3469884395599365, 0.44944724440574646, -2.706183671951294, -0.5161762237548828, -0.1955806165933609, -0.3711612820625305, 0.0049304901622235775, 1.913141131401062, 1.7575129270553589, 0.7744871377944946, -0.29306530952453613, -1.0919374227523804, 0.07355771213769913, -0.8726858496665955, -0.09946359694004059, 0.26126953959465027, 2.8298535346984863, 0.6714100241661072, -0.9087213277816772, 0.05022136867046356, -0.29825153946876526, 0.7513856291770935, 1.03369140625, -0.29941415786743164, -0.3349582552909851, -0.8120465278625488, 1.5006033182144165, 0.26509883999824524, -0.6397119164466858, 1.8078365325927734, -0.007823165506124496, -0.6740851998329163, -1.2863430976867676, -0.6167706847190857, -0.9015285968780518, -0.15089550614356995, -0.04508016258478165, -1.1818714141845703, 1.4915319681167603, 0.5162193775177002, -0.33919116854667664, -0.14365987479686737, 1.1223725080490112, -0.5523073077201843, 1.6196584701538086, -0.24200895428657532, -0.8184183835983276, 0.6335077881813049, 1.2236266136169434, -0.3005315661430359, 0.39337924122810364, 0.42118456959724426, 1.0234367847442627, 0.618061363697052, -1.6755220890045166, 0.9867550134658813, -0.422149121761322, -1.0164015293121338, 0.29559171199798584, -0.5675992965698242, -1.0797849893569946, 0.9178088903427124, -1.4126582145690918, 0.8015571236610413, 0.4839235246181488, 0.8053669333457947, -1.395574688911438, 0.11174960434436798, -0.7471916079521179, -0.17210634052753448, -0.6519132852554321, -0.33932262659072876, 0.331127405166626, 0.784430205821991, 0.9669327735900879, -1.1811342239379883, 1.0187175273895264, 1.4531456232070923, -1.0180597305297852, -0.5799934267997742, 1.9517810344696045, 0.17909595370292664, 1.485393762588501, 0.36560526490211487, -0.9917495250701904, -0.021988818421959877, -0.4396221339702606, -0.17902177572250366, 0.6032326817512512, -0.02508525177836418, 0.9986236095428467, 0.40559840202331543, 0.390198290348053, -0.32485464215278625, -0.08399451524019241, 0.984472393989563, 1.1647684574127197, 1.3043270111083984, -0.1685451716184616, -0.38723400235176086, -1.4575127363204956, 1.4966987371444702, -0.16976645588874817, 0.5324052572250366, 0.7903522849082947, 0.46152934432029724, 0.860732913017273, 0.9213306903839111, 0.35681259632110596, 0.7344946265220642, 1.7402223348617554, 1.7277647256851196, 0.43415138125419617, -0.059802837669849396, -0.4849281311035156, -1.4640592336654663, 0.15317584574222565, 0.19822484254837036], [0.8744598031044006, 0.14183150231838226, 1.2126266956329346, -1.2013978958129883, 0.06624893844127655, -0.08385401219129562, 0.5707110166549683, 0.24847708642482758, 0.37461981177330017, 1.6554524898529053, 1.8442726135253906, -1.1603816747665405, 0.5635634660720825, -1.4475375413894653, 1.491501808166504, -0.15400543808937073, 1.2857164144515991, -0.2792331278324127, -0.4454982280731201, 0.8187977075576782, -0.7200029492378235, 0.2979460656642914, -0.05385737866163254, -0.6097978949546814, 1.1515074968338013, 0.6389193534851074, -0.26086515188217163, 0.3005134165287018, -1.295318841934204, -1.3766800165176392, -1.0744655132293701, 1.2155770063400269, 0.4876847565174103, -0.4292871057987213, 2.085130453109741, 1.9855469465255737, 0.2855030596256256, -0.1542457789182663, -0.7307377457618713, -0.8877537846565247, 0.49472033977508545, -1.1767131090164185, 0.5801167488098145, -0.051782507449388504, -0.44178009033203125, 1.7740020751953125, 0.4920386075973511, 0.11350374668836594, -0.3345125615596771, -1.0897465944290161, -0.9980124235153198, -0.49252521991729736, -0.17112620174884796, 0.1362423151731491, -0.3836490511894226, 0.870789110660553, 0.7714040279388428, 0.763580322265625, -1.484119176864624, 1.4231234788894653, -1.2350071668624878, -0.39155372977256775, -0.6586464643478394, -3.436370611190796, -1.3384979963302612, -0.3292049467563629, -0.9886521100997925, -0.3787255585193634, 0.5309578776359558, -0.08665972203016281, -1.0317820310592651, -0.8659258484840393, -0.027606099843978882, 2.3110220432281494, 0.2804272770881653, -1.4700754880905151, -0.12520678341388702, -0.13977956771850586, 0.09203166514635086, 0.6447827219963074, -0.38986170291900635, 0.9869094491004944, 0.5147291421890259, -0.4156312942504883, 0.28200721740722656, 1.2933207750320435, -1.211423635482788, 0.18217796087265015, 0.9556325078010559, 0.444364070892334, 0.6407495141029358, -2.017402410507202, 2.435264825820923, 1.0376896858215332, 1.2963205575942993, 0.17640851438045502, -0.42138758301734924, -0.47894957661628723, 0.38420620560646057, -0.8802123069763184, 0.4671643376350403, 2.8473222255706787, -0.3799363672733307, 0.7766370177268982, 0.24969838559627533, -1.5111865997314453, 0.25291410088539124, -1.503721833229065, -2.262381076812744, 0.09280361235141754, 2.415539026260376, -0.024698909372091293, 0.4188784956932068, -0.5536078810691833, -0.6671620607376099, -0.07814500480890274, -0.8878389596939087, 1.824457049369812, -0.583995521068573, -0.8266393542289734, -0.9037846922874451, 1.5186777114868164, 1.7950005531311035, 1.097961664199829, -0.7377866506576538, 1.3059132099151611, -1.0207479000091553, 1.0334640741348267], [1.4374825954437256, -0.574453592300415, 0.8633168339729309, 0.7754034996032715, -0.660129189491272, -1.3101494312286377, -0.3696020841598511, 1.568922758102417, 1.6162039041519165, 0.09845668077468872, -0.48472657799720764, 0.03776749223470688, -0.9008498191833496, -1.3641983270645142, 0.75816810131073, -1.0457675457000732, 0.6776201725006104, -1.285532832145691, -1.1065118312835693, -0.5564354658126831, -0.2943045496940613, 1.246104121208191, 0.05724004656076431, -1.006136417388916, -1.6468734741210938, -0.4413450360298157, -0.5422475934028625, 1.2089484930038452, -0.15998922288417816, -0.9840403199195862, 0.8187069892883301, 0.2912885844707489, 1.1156525611877441, -1.105383038520813, -1.912556529045105, 1.5508615970611572, -1.7072105407714844, 1.3376573324203491, 1.3186243772506714, 2.1488394737243652, -0.2029801607131958, 1.5323196649551392, 1.2125580310821533, 1.1411782503128052, 1.1715542078018188, -0.06689909100532532, 1.6262716054916382, 1.2875370979309082, -0.1646469086408615, 0.24898582696914673, -1.3402323722839355, -0.6533209681510925, 1.7705668210983276, -0.9041174054145813, -1.9845913648605347, 0.26711326837539673, 1.0171433687210083, -0.6527009010314941, -1.0294874906539917, -1.4375871419906616, -0.3581988215446472, -1.1731598377227783, -0.3230791389942169, 0.45038530230522156, -0.6362849473953247, -0.7802286744117737, -0.7826669216156006, 0.23180079460144043, 0.25078800320625305, 0.06618111580610275, -0.7718246579170227, 0.3229278028011322, 2.0670900344848633, 1.0626405477523804, 0.7641012668609619, 2.9019851684570312, 0.611305832862854, -0.26749494671821594, 0.5684085488319397, -0.44021016359329224, -0.6648436784744263, 0.5298272967338562, 0.2932334840297699, -0.5067629814147949, 1.6322461366653442, -2.060947895050049, -0.4378184974193573, 0.7775707840919495, 0.5244945287704468, -0.9639666080474854, 0.2421625554561615, -1.0382598638534546, -0.804384171962738, 1.6531031131744385, 1.1725152730941772, 0.028038306161761284, 0.9901683926582336, 0.3865627348423004, 1.4780263900756836, 0.09130120277404785, -1.1348718404769897, 0.6504960656166077, -0.5807345509529114, 0.8410465717315674, -1.666364312171936, 0.13488908112049103, 1.2362703084945679, 0.8764377236366272, -0.5459675192832947, 0.5266079902648926, -0.9578190445899963, -0.632973849773407, -1.1608377695083618, -0.5396175384521484, -0.1352996826171875, -0.9501007795333862, 0.5691216588020325, -0.9353757500648499, 0.693694531917572, -0.38217881321907043, 1.1395092010498047, -0.32099053263664246, 0.01678820699453354, 0.9652231335639954, -0.277145117521286, -0.944614827632904, 0.6191005706787109, 0.37214866280555725], [1.0344271659851074, 0.33727285265922546, -0.690311849117279, -0.20373597741127014, 0.8255988955497742, 0.24279829859733582, -0.6409341096878052, -0.3003085255622864, -0.6049727201461792, -0.6859170198440552, -0.40260443091392517, 1.1502712965011597, -0.23661565780639648, -1.0288852453231812, -1.0310362577438354, 0.7473437190055847, -0.34481117129325867, 1.7823371887207031, -1.5614439249038696, -0.4299772083759308, -1.5612446069717407, 0.28392815589904785, 1.13560950756073, 0.8358489274978638, -0.017579803243279457, -1.1275535821914673, 1.4726454019546509, 0.06673812121152878, 0.1681557148694992, -1.8495121002197266, 1.9529026746749878, -0.49381351470947266, -1.815507411956787, -0.3243522644042969, 0.4128056764602661, 0.344144344329834, -0.2742269039154053, -0.4561605751514435, 0.6606107354164124, 0.9064153432846069, -0.648047685623169, -0.1580141931772232, -0.9732648730278015, -0.2924664318561554, -1.5370843410491943, 0.38830050826072693, -1.9611916542053223, -0.23519086837768555, -0.1759405881166458, -0.8385758996009827, -1.0244216918945312, -0.5289813280105591, -0.07049427181482315, -2.04303240776062, 1.183236002922058, 1.6937754154205322, 0.7919897437095642, 0.9430707097053528, 0.36046162247657776, 0.9317947626113892, 1.7372697591781616, 0.6928703784942627, -0.08918885141611099, 1.1349132061004639, 1.2121881246566772, 0.9886146187782288, -0.5219687223434448, -0.6585291624069214, -1.1552175283432007, 0.7433043718338013, -1.336444616317749, 0.7955812811851501, 1.2348179817199707, 0.20959196984767914, 0.0658283531665802, 0.9055044054985046, -1.1967321634292603, 0.42866024374961853, -0.7699609994888306, -1.4988733530044556, 1.3542474508285522, -0.047367196530103683, -1.302756905555725, 0.501127302646637, 0.5750049948692322, -0.6242141127586365, 1.10995614528656, 1.6700071096420288, -0.43032124638557434, 0.17862941324710846, -0.5518516302108765, -0.23736710846424103, 1.1414377689361572, 0.4009106159210205, -0.5641553997993469, -1.1122426986694336, 1.1780219078063965, -0.14958544075489044, 1.88251531124115, -0.6744577288627625, 0.3358865976333618, -0.3752298057079315, -2.073990821838379, -0.8997208476066589, -0.13629867136478424, -1.579176902770996, 0.021501967683434486, -1.5523570775985718, 0.7534142732620239, -0.24515396356582642, -2.0322508811950684, -0.09090140461921692, 0.5011346340179443, -0.41590604186058044, -1.3753931522369385, -1.0289287567138672, 1.3910242319107056, -0.039408378303050995, 2.3992767333984375, -0.010637094266712666, 1.2905670404434204, 1.5987352132797241, -0.508399248123169, -1.1628644466400146, -0.13457359373569489, 1.4050425291061401, -0.07156286388635635, 0.19809377193450928]], [[0.7357699275016785, 1.0904830694198608, -0.03110329806804657, -0.2440067082643509, -0.7500715255737305, 0.8548301458358765, -0.47307097911834717, 0.37417900562286377, -0.7408764958381653, -0.05456779524683952, -0.9821315407752991, 0.2503346800804138, 0.2031196504831314, -0.9373407363891602, -0.4459565579891205, -1.1164416074752808, -0.6686015725135803, 0.23713237047195435, -0.16707095503807068, -0.5501143336296082, -0.36715927720069885, 0.13603152334690094, -0.019841961562633514, 0.7425075173377991, 0.551895260810852, -0.1310945749282837, -0.5705747604370117, -1.0355963706970215, -0.7079426646232605, -1.0734399557113647, -2.471585750579834, 1.2504143714904785, -0.49551549553871155, -2.215627908706665, -1.007020115852356, -1.8747258186340332, 1.8569567203521729, -0.7710604071617126, -1.910239815711975, 1.005190134048462, -1.1435788869857788, -1.6729154586791992, 0.31840768456459045, -1.3239407539367676, 0.678330659866333, -0.43894174695014954, -0.18502050638198853, -0.18550212681293488, -0.22166745364665985, 0.08529116958379745, 1.342283010482788, 1.035774827003479, 1.0185351371765137, -0.2874176502227783, -1.6801369190216064, -0.6837306618690491, 0.5319011211395264, 1.533512830734253, 0.7764478325843811, 0.7325175404548645, 2.7793233394622803, 1.3313663005828857, 1.2972655296325684, -0.11173848807811737, 0.2830376923084259, 1.7536616325378418, 0.7329756021499634, -0.8542625308036804, -0.07838529348373413, 0.32823729515075684, -1.1645383834838867, -0.022218389436602592, -0.18207111954689026, 0.9338870048522949, -1.983712911605835, -0.5123054385185242, -0.7872235178947449, -0.979366660118103, 1.172134518623352, 0.7644405364990234, -0.21636618673801422, 0.14567047357559204, -0.08128238469362259, -0.4498523771762848, -1.047218680381775, -0.33428263664245605, -0.20402753353118896, 1.0628297328948975, -0.5033755898475647, -1.674970269203186, 0.6813990473747253, 0.6394944190979004, -0.6449286937713623, -0.692521333694458, 1.1768535375595093, 0.6321377158164978, -1.3297629356384277, 0.7157272696495056, 0.2667594254016876, 0.45217734575271606, 0.27487578988075256, -0.6339762806892395, -0.632569432258606, 0.3710382580757141, -1.0962226390838623, -0.575534999370575, -0.9321308135986328, -1.156416654586792, 0.17084327340126038, 0.20721617341041565, -0.09369999170303345, -1.6332635879516602, -0.8329198360443115, -0.4293510317802429, -0.1885569542646408, -0.5987759828567505, 2.2198033332824707, 0.9048826098442078, 2.012489080429077, -0.5688177347183228, -0.213300421833992, 0.7364381551742554, -0.8103317022323608, -0.5942371487617493, 0.7053214311599731, -1.316866159439087, 2.2854764461517334, 0.7276705503463745], [-0.21667197346687317, -1.4982364177703857, 1.1573065519332886, -1.8326376676559448, 0.9344457387924194, -0.052143391221761703, 1.5652904510498047, 0.078682541847229, 1.5514706373214722, -0.5745064616203308, 0.35670074820518494, 0.23292046785354614, 1.269511342048645, 0.47886931896209717, -1.4324471950531006, -0.09068839251995087, -0.35728082060813904, 0.5355604887008667, -1.708418846130371, 0.5508553385734558, 0.6624270677566528, -0.08977367728948593, -0.16126205027103424, -0.5027607679367065, -0.2522030174732208, 0.9362190365791321, 0.8580822348594666, 0.17861954867839813, 0.30062368512153625, 0.7095088362693787, 0.19295941293239594, -0.5358613729476929, 0.5027425289154053, -0.6315310597419739, 0.5483578443527222, -0.021777868270874023, -0.3680333197116852, 0.8856121897697449, -1.3765469789505005, -0.3804906904697418, -0.4378712475299835, 0.47905129194259644, 1.6110347509384155, -0.014578523114323616, -0.6309909224510193, -1.3753947019577026, -1.8654346466064453, 0.929226279258728, 1.1642659902572632, 0.26404672861099243, -0.15861552953720093, -0.08115965127944946, 0.3641672432422638, -0.05898623540997505, -2.14520001411438, 1.8476083278656006, -0.3528803586959839, 0.8571426868438721, 0.602358877658844, 1.0729737281799316, -0.9407114386558533, -0.9443584680557251, 2.2330687046051025, 0.13846665620803833, 1.0306819677352905, -0.8109065890312195, -1.3994845151901245, 0.47944751381874084, -1.069778323173523, 0.6419216990470886, 0.19814151525497437, 0.8195075988769531, -0.47260528802871704, 0.5989230871200562, -0.24301022291183472, -0.4018992483615875, 0.11804737895727158, -0.14852669835090637, -0.07741352170705795, 1.6016947031021118, 0.21780110895633698, 1.2924349308013916, -0.7382308840751648, -2.44918155670166, -0.7485328316688538, 0.519987165927887, 0.24944961071014404, 0.5109305381774902, 1.28333580493927, -0.9269894361495972, 0.621631383895874, 0.502648115158081, -0.0856735110282898, 1.7677583694458008, -0.6351884007453918, 0.5821057558059692, -1.4520039558410645, 1.1866681575775146, -1.5464197397232056, 0.7160801291465759, 1.1887038946151733, -0.8117077350616455, -1.5910046100616455, -1.5032232999801636, -0.6420842409133911, 0.022651419043540955, 0.41801121830940247, -0.16367091238498688, -0.05706227943301201, -0.17155082523822784, 0.47855380177497864, 0.4388306736946106, -0.4133893847465515, 0.31325656175613403, 0.7962421178817749, 0.48582661151885986, -0.5133364200592041, 0.2313324362039566, -0.7593621611595154, -0.5981859564781189, -1.1800440549850464, 0.24464499950408936, 0.3801167905330658, -1.6162679195404053, 0.3714723587036133, 0.01535211130976677, -1.5316702127456665, -1.5304712057113647], [-0.21499153971672058, -0.37589868903160095, 0.36480262875556946, -0.6623382568359375, -0.8572933673858643, -0.4748615026473999, -0.11328766494989395, 1.2090821266174316, -0.2136089950799942, 0.05110989138484001, -0.8486298322677612, 0.19468161463737488, -0.899580180644989, 1.0786136388778687, 0.6073864698410034, 1.6290339231491089, -1.0690820217132568, 1.1931064128875732, -0.5624408721923828, -0.007851239293813705, -1.2877743244171143, 0.8242811560630798, -1.0135389566421509, 0.13527071475982666, -1.6977177858352661, -1.4478658437728882, 0.1378214806318283, -0.14745110273361206, -0.07991506904363632, 2.4036192893981934, -0.47199270129203796, -0.9301869869232178, 1.8252202272415161, -0.010938052088022232, -1.2856471538543701, -0.6576066017150879, -0.18953044712543488, -1.054380178451538, -1.2470453977584839, -0.5180413126945496, 1.5269098281860352, 1.6813722848892212, -1.5089619159698486, 0.1017148494720459, 1.3497709035873413, 1.3858585357666016, -1.6747902631759644, 0.29625314474105835, 0.30873599648475647, -0.10592266917228699, -0.40801337361335754, -0.03347482532262802, 0.011064905673265457, -1.537006139755249, 1.6347661018371582, -0.37073060870170593, -0.38085880875587463, 1.4233160018920898, 0.43474188446998596, 1.749423623085022, 2.819269895553589, 0.11679580062627792, 0.6262811422348022, -0.6424574255943298, -0.1722387671470642, -0.7534509301185608, 0.8172225952148438, 0.7101467847824097, 1.759626865386963, 0.0838392898440361, 1.037529468536377, 0.9862056970596313, 0.2580699622631073, -1.1363022327423096, 1.0030651092529297, -0.47256895899772644, 0.17929242551326752, 0.7296721935272217, -0.5070073008537292, 0.5911109447479248, -0.6477540731430054, -0.538878858089447, -0.7851719260215759, 0.1779886931180954, -0.05939965695142746, 1.2115613222122192, 0.2848931550979614, 1.145115613937378, 0.366521954536438, -1.0336570739746094, 0.03785174340009689, 1.473325490951538, 0.8882237076759338, -1.0848227739334106, -1.0056591033935547, -0.3713291585445404, 1.1548011302947998, 1.5227831602096558, -0.7955714464187622, -0.504295289516449, 0.2675340175628662, 1.3831384181976318, -1.5805734395980835, -0.9412325024604797, -0.1822347193956375, -0.04175037145614624, -1.1280981302261353, -0.2636929452419281, 0.6977700591087341, 0.10289429873228073, -1.298933744430542, 0.7595588564872742, -0.7999964952468872, -0.3444979190826416, -2.2879862785339355, 0.7340718507766724, 0.7018710970878601, 0.8758835196495056, 0.32883134484291077, -0.2066284716129303, 0.42671093344688416, -0.6298395991325378, 0.4849950969219208, 0.0755828395485878, 0.1782972812652588, -0.22473646700382233, -0.05020894482731819, -0.6232871413230896], [-0.7672621607780457, 0.7920758128166199, -0.3343486487865448, -1.4473676681518555, -1.1776413917541504, -0.9435350298881531, -0.3087024390697479, -1.3240858316421509, -0.47385066747665405, 0.06706048548221588, 1.4533469676971436, 1.0240850448608398, 0.9155077338218689, 0.5488210320472717, 0.35411569476127625, 1.6948485374450684, 2.3290913105010986, -0.9357835650444031, 0.11074844747781754, -0.5981621146202087, -0.08775532990694046, -1.0862208604812622, -1.7074073553085327, -1.6378653049468994, 0.037707533687353134, -0.9740980267524719, -0.10460428148508072, 0.30097082257270813, -1.0380945205688477, 0.8027545213699341, 0.6529077887535095, -0.11631356924772263, -1.2523775100708008, -1.5872551202774048, -1.3880412578582764, 0.605209469795227, 1.240347146987915, -0.6626086235046387, -1.8325773477554321, 0.4247191846370697, -0.21249915659427643, 0.7631049752235413, 0.22593623399734497, -0.31253471970558167, 0.44965407252311707, 0.3906552195549011, -0.48306557536125183, -1.1633093357086182, 2.155402421951294, 1.4378879070281982, 0.0055229938589036465, 0.6084895730018616, 0.7691371440887451, -2.8471732139587402, -0.17688144743442535, 0.14473192393779755, -1.8153961896896362, 0.8294507265090942, -1.7420017719268799, -1.2664241790771484, 0.2788275182247162, 0.6019247174263, 1.1085602045059204, -0.5842787623405457, -0.9650906920433044, 0.5532054901123047, -0.8737550973892212, -0.3250502943992615, 0.2581879794597626, -1.2963391542434692, -0.5484864711761475, 1.366034984588623, 0.4338098168373108, 0.21904847025871277, 0.06427119672298431, 0.4049180746078491, -0.09838835150003433, -0.5108232498168945, -0.26274511218070984, 0.3796849846839905, -1.3151756525039673, 1.643480658531189, -0.13728965818881989, -0.8201581835746765, -0.07565434277057648, 2.046268939971924, -0.6110166311264038, -0.5749373435974121, 0.30199581384658813, -0.5562816262245178, 1.515735149383545, -0.9307144284248352, -0.047420043498277664, 0.9653275609016418, 0.3386927545070648, 1.4813176393508911, -0.949451744556427, -1.9666593074798584, 0.0339336022734642, 0.03436415269970894, 1.8954719305038452, 0.3960736393928528, 1.5373454093933105, 0.1572318971157074, 0.6078492403030396, -0.7976164817810059, 0.04721124470233917, -0.21240366995334625, 0.4843396842479706, -0.8810673356056213, -0.9154780507087708, -1.1459007263183594, -0.310456782579422, -0.9512798190116882, 0.7699844837188721, 0.3335041403770447, -1.1256788969039917, 0.17045477032661438, 0.6431204676628113, -2.426497220993042, 0.004571934230625629, 0.2761671841144562, -0.08466416597366333, -0.7016350626945496, -1.1748994588851929, 1.0169544219970703, 0.054380737245082855, 0.9391920566558838], [1.4712917804718018, 1.228007197380066, -1.9264110326766968, 1.561949372291565, 0.3952578902244568, 0.8852284550666809, -1.4104437828063965, -1.3875328302383423, 0.56144779920578, -0.16890272498130798, 0.4632323086261749, 0.4849758744239807, 0.6754207611083984, 1.1928753852844238, 0.4301126003265381, 0.6258671879768372, 1.6106237173080444, 1.1491116285324097, -1.638720154762268, -0.5988110899925232, 0.5633618235588074, 1.984600305557251, 1.6862914562225342, -0.7970185279846191, -0.017086608335375786, -0.8299508094787598, 1.5393303632736206, 2.2166032791137695, -1.1239746809005737, 0.8004220724105835, -1.4060959815979004, -0.17536351084709167, 2.247100591659546, -1.0090426206588745, 0.5583533644676208, -0.3171921372413635, 2.181692361831665, 1.3495944738388062, 0.09815158694982529, 0.8954665064811707, -0.5946269631385803, 0.4571567475795746, 0.23074384033679962, -0.343680202960968, -1.2571439743041992, 1.193963646888733, -0.715183675289154, 0.9801735281944275, -0.21128053963184357, -0.8346067667007446, -0.251603901386261, -0.9535520076751709, -0.3732675313949585, -0.07879116386175156, 0.5920276045799255, 0.33048972487449646, 0.2728235423564911, -1.485854148864746, 0.21572551131248474, 0.13134917616844177, -0.041598841547966, 1.242068886756897, -1.7405650615692139, 0.4512943923473358, 0.8038247227668762, -0.46500757336616516, 1.2758302688598633, 2.2694509029388428, 0.45966124534606934, 0.7199098467826843, 0.38986265659332275, 1.1205447912216187, -0.01610645279288292, -0.1358117312192917, 1.0392358303070068, -0.9353829622268677, 0.28584516048431396, 0.5143327713012695, -2.1869254112243652, -0.655604362487793, 0.3876691460609436, -1.5044350624084473, -1.2979472875595093, -1.4016774892807007, -1.6436047554016113, 0.6207820177078247, 0.09855901449918747, 0.9596302509307861, -0.760512113571167, -0.9733083844184875, -1.7276970148086548, -0.4410133957862854, -0.4677557647228241, -0.2577035129070282, -1.7914152145385742, 0.9823541045188904, -0.3743736147880554, -0.0033297017216682434, -0.01068541407585144, -0.7927822470664978, 0.8714592456817627, 0.4409175217151642, 0.05337359011173248, 1.4168879985809326, -0.21388299763202667, 0.6780398488044739, -0.3803437352180481, -0.1295144259929657, -1.430099606513977, 0.4165589511394501, 0.7193570137023926, -0.742953896522522, -0.19712843000888824, 0.2780148684978485, 0.24069954454898834, -0.16252878308296204, -0.7730544209480286, -0.06218560039997101, 0.3202528655529022, 0.8072916865348816, -0.6024262309074402, -1.5513403415679932, 0.30939772725105286, -0.11902624368667603, -0.6712594628334045, 0.8566460609436035, -2.270343065261841, 1.0179194211959839], [0.8678141832351685, 0.6102228760719299, 0.47815442085266113, 0.3255605697631836, -0.3262670040130615, 1.0662140846252441, -0.3199586272239685, -0.7421023845672607, -0.8938431143760681, -0.3790692687034607, -0.5054024457931519, -0.9819528460502625, -0.8088545203208923, -0.49236202239990234, 0.4969423711299896, 0.5151785612106323, 0.045899756252765656, -1.066981554031372, -2.4069371223449707, -0.9572281241416931, -0.48329460620880127, 1.9419759511947632, -0.27432024478912354, 0.06294135749340057, -0.9445936679840088, -0.43276864290237427, -1.1611404418945312, -0.36485201120376587, -1.0251407623291016, 0.07132913172245026, -0.7808430790901184, 0.601526141166687, 1.1952213048934937, 0.3559022545814514, -1.16942298412323, -0.5184974074363708, -0.12827496230602264, -0.886860728263855, -0.44839033484458923, 0.4090118110179901, -0.846302330493927, 1.9405725002288818, -0.7044844031333923, -0.1905783861875534, 0.2469828575849533, 0.07697148621082306, 1.9369823932647705, 1.0748226642608643, 0.8264045715332031, 0.735159158706665, 0.2802676260471344, -0.7955257892608643, 0.5056875348091125, 0.10143061727285385, -1.0600433349609375, 0.1622985303401947, 1.045936107635498, -0.05277867242693901, -1.933205246925354, 1.7691843509674072, 0.08284562826156616, 1.260591745376587, 0.35258403420448303, 1.6134546995162964, 1.7467268705368042, 0.5665335059165955, -0.09853067249059677, -0.6761158108711243, -1.0027155876159668, -0.11298616975545883, 0.39314568042755127, 0.78481525182724, -0.4240347445011139, 0.10466225445270538, 0.06568888574838638, 0.03807449713349342, -1.8925604820251465, 0.735242486000061, 0.7310463786125183, -0.0999516174197197, 0.5930948257446289, -0.597805380821228, 1.1185749769210815, 0.707107663154602, -0.18151548504829407, 1.2175755500793457, -0.061082638800144196, 0.6380906105041504, 0.502998948097229, -0.1316649615764618, 0.07710005342960358, -0.9768017530441284, 0.9114539623260498, -0.5114547610282898, 0.6991387009620667, -1.2890833616256714, 0.2725520730018616, 0.515815794467926, 0.9946267008781433, 0.07237087190151215, -0.5707104206085205, -0.5228750705718994, -0.0713188573718071, 1.0265482664108276, -2.871929407119751, 1.8135203123092651, 0.7152958512306213, -0.06181924417614937, 2.3221259117126465, 1.6356406211853027, -0.060637056827545166, 0.9716554284095764, 0.20679859817028046, 0.19759055972099304, 0.7280850410461426, -0.13144010305404663, 0.8434886336326599, 2.340050220489502, -1.2841359376907349, -1.2571865320205688, 0.33241167664527893, 1.302995204925537, -1.7938807010650635, 0.22400856018066406, 0.5659070611000061, 0.5996412634849548, -0.33220481872558594, 2.0059309005737305], [1.160386085510254, 0.551159679889679, 1.5331684350967407, 1.2904510498046875, -0.04728453606367111, 0.11420492827892303, -0.939450740814209, -0.24055010080337524, -1.469459056854248, -0.3896346390247345, 0.10121804475784302, 0.15678709745407104, 0.7316163778305054, -1.2990480661392212, -0.3389241099357605, -0.19648601114749908, -0.7857059836387634, -2.474781036376953, -1.6429474353790283, -2.5671093463897705, 1.880438208580017, -0.6290431618690491, 0.43209755420684814, -0.48773542046546936, -0.7402839064598083, -1.2896555662155151, 0.8827189803123474, 1.1141573190689087, 1.1548919677734375, 1.1097369194030762, -1.4954475164413452, 0.7254267930984497, -0.5690135955810547, 1.093239188194275, -0.1971811056137085, -0.21781067550182343, 0.5511006712913513, -0.5639526844024658, -0.5136721134185791, 0.7284673452377319, -0.030816202983260155, -1.2221299409866333, 0.04260030761361122, -1.0098479986190796, -1.1784170866012573, -1.0442594289779663, 1.0678956508636475, -0.2133679836988449, -0.7770057320594788, 0.28731369972229004, 1.4648770093917847, -0.6291236281394958, -0.7336247563362122, 0.33040863275527954, -0.3604060113430023, 1.5030523538589478, -0.19873173534870148, 0.19167816638946533, -1.754050374031067, 0.12267675995826721, -0.07159776240587234, -3.167224645614624, -1.989223837852478, 0.5725073218345642, -0.9968472719192505, -0.34559163451194763, -0.686371386051178, -0.7060654759407043, -0.16181381046772003, 0.48236650228500366, -0.11878208816051483, 0.3849036395549774, 0.25237053632736206, 0.7423004508018494, -0.4828297197818756, 1.8502284288406372, 0.005744041409343481, 2.7833073139190674, 1.1944462060928345, -0.04055139422416687, -0.3090095818042755, 0.18097788095474243, -1.5468316078186035, 1.0755881071090698, 0.1082935631275177, 1.1868572235107422, -0.858586311340332, 1.1780847311019897, 0.9032089114189148, 0.029540153220295906, 1.3594729900360107, -0.02711435593664646, -0.3646317422389984, -0.17767392098903656, 0.7898431420326233, -1.3844780921936035, -0.09276995062828064, 2.0071611404418945, -0.5258895754814148, 1.0857573747634888, -0.8592900633811951, -0.7325100302696228, 0.21762292087078094, 0.7653759121894836, 0.4170992970466614, 0.6765270829200745, 1.0437321662902832, 0.18435890972614288, -0.5692761540412903, -0.8069449067115784, 0.13916276395320892, -1.1479283571243286, 0.6639373898506165, -0.7481284141540527, 0.31810539960861206, -0.5325844883918762, 0.04735066369175911, -1.0948926210403442, 1.075990080833435, -0.380719929933548, -0.6027491092681885, -1.7757518291473389, -1.7533271312713623, -0.6594346761703491, -1.7261877059936523, 0.5887208580970764, -1.0070244073867798, -0.4220935106277466], [-0.15230949223041534, -0.12397617846727371, 1.2152835130691528, 0.291258305311203, -0.16516606509685516, 0.6762241721153259, 0.6411278247833252, -1.0697789192199707, -0.46540573239326477, -0.5483875274658203, -0.3981491029262543, -0.1984715759754181, 1.0354353189468384, -0.6478352546691895, -0.3156134784221649, -0.19188764691352844, 1.4522167444229126, -0.7852703332901001, 0.22869873046875, -0.15586410462856293, -0.5692543983459473, -0.2381134033203125, 0.14703300595283508, 2.1103248596191406, 0.12343473732471466, 0.27213752269744873, 0.006160991732031107, 0.3662371337413788, 2.081058979034424, 0.3835207521915436, -1.2131109237670898, -0.7866694927215576, -0.5515573024749756, 1.2314462661743164, 0.1904875487089157, -1.7243683338165283, 0.20536643266677856, -2.176129102706909, -0.02928897924721241, -0.2377062439918518, 0.683018147945404, -0.8551360368728638, 1.1234972476959229, -0.23143160343170166, -0.6040008664131165, -0.5306875705718994, 0.26086875796318054, 0.2838079631328583, -0.9387985467910767, 0.4789108335971832, 0.9092509746551514, -1.007232666015625, -0.16806429624557495, 0.27381452918052673, 0.5230271816253662, -0.23830769956111908, 1.6897552013397217, 1.2708626985549927, 0.21648819744586945, -0.7013816237449646, 0.9255144596099854, -0.3522382080554962, 0.4737650752067566, -1.261213779449463, 0.500798761844635, -1.6845650672912598, 0.08282078802585602, -0.2727496325969696, 0.7461093664169312, 0.27439868450164795, 0.08745261281728745, 1.6175642013549805, 0.13389013707637787, -0.8945814967155457, 0.49994587898254395, -0.23084606230258942, 1.3386656045913696, 0.586076021194458, -2.3904542922973633, -0.7631139159202576, -1.482740044593811, 0.3121435046195984, 1.8246476650238037, 1.1013784408569336, -0.4146876931190491, 0.2336399108171463, 0.48389747738838196, -0.7627175450325012, -1.1198694705963135, 0.1397174447774887, 0.9622180461883545, 0.23293063044548035, 0.44286492466926575, -1.9230648279190063, 0.37234795093536377, 0.5030712485313416, -1.0280451774597168, 0.2586556673049927, 0.317840576171875, -1.2473652362823486, -0.37589263916015625, -0.12960900366306305, 0.2857513129711151, 1.1280338764190674, -1.0242422819137573, 0.44233959913253784, 1.1748266220092773, 1.1433645486831665, 0.5612683296203613, 0.3821650743484497, 0.013905242085456848, -0.8943567276000977, 1.3101128339767456, 0.25085675716400146, -0.1567549854516983, -0.5176668167114258, -1.49082350730896, -0.3668157458305359, -0.9130109548568726, 0.7125357389450073, -1.4013152122497559, 0.22679460048675537, 1.3625003099441528, 0.3383302688598633, -0.2687436044216156, 2.33735728263855, 0.00721735879778862, -0.23625126481056213], [-1.0145212411880493, 0.06567583233118057, -1.764407992362976, -0.9451361894607544, 0.6656559109687805, -0.4248507618904114, -0.7846670746803284, 1.7381436824798584, -0.3016063868999481, 1.0678346157073975, -0.6501278281211853, -0.9467922449111938, 0.15833702683448792, -0.49764829874038696, -0.7415640950202942, 1.832688570022583, 0.9975652694702148, -0.5330215692520142, 0.9926212430000305, 0.3450097143650055, -2.190159559249878, -1.1192858219146729, 0.4134390652179718, 1.2208149433135986, 0.2527925372123718, -0.17501363158226013, -0.29104146361351013, 0.02355526015162468, -1.5002561807632446, -0.3834519386291504, -0.24854956567287445, 0.8209061026573181, -1.0695995092391968, 1.287840485572815, 1.5295051336288452, 0.7680138945579529, -1.8644657135009766, 1.3794618844985962, -0.7404347658157349, -0.20674386620521545, -0.476961225271225, 0.5456463098526001, -0.6622015833854675, -1.3549751043319702, 0.7478327751159668, -1.893314242362976, 1.414713740348816, -0.7726339101791382, 1.1186894178390503, 0.5318055748939514, 0.17261871695518494, 1.051914930343628, 0.08618389815092087, -0.5724254250526428, -0.12073881179094315, 0.8157456517219543, -1.7569068670272827, 0.11877737939357758, 0.20230671763420105, -0.8098766803741455, 0.8075235486030579, 0.41146454215049744, 0.31497740745544434, -0.64427649974823, 0.9372661113739014, -0.5673393607139587, 1.6933420896530151, -1.1295744180679321, 1.1514942646026611, 0.6056162714958191, -0.6044449210166931, -0.1845542937517166, 0.833652138710022, -0.11261186003684998, -0.5780101418495178, 0.321098655462265, 1.4656518697738647, -1.4360548257827759, -0.3393973410129547, -0.41096457839012146, 0.6992623805999756, 1.3644487857818604, -0.9131103157997131, 0.5227442383766174, 0.7622100710868835, -1.1413532495498657, -1.7037913799285889, 0.946560800075531, -0.5948725342750549, 0.3697458803653717, 0.33996331691741943, 1.2912896871566772, -0.3571337163448334, 1.180791974067688, 1.1654167175292969, 1.946302056312561, -0.1932554394006729, 1.3629353046417236, -0.007866710424423218, 0.9264146089553833, -0.27375301718711853, 1.247009515762329, -0.8001329898834229, 0.6827213764190674, 0.7980422377586365, 1.0677069425582886, 0.8614432215690613, 0.409479558467865, 1.6554783582687378, -0.2608407139778137, 0.007626499980688095, -1.784169316291809, 0.2562713027000427, 0.5979797840118408, -1.3611223697662354, -1.1072391271591187, -2.8066565990448, 0.558746874332428, 0.05999220162630081, 1.9300998449325562, 0.5327141284942627, -0.20980346202850342, 2.789750814437866, -0.037852171808481216, 0.9671624898910522, 1.2319531440734863, 0.27946051955223083, 0.2908802032470703], [1.900665283203125, -1.548905372619629, 1.2724878787994385, 1.1578134298324585, 0.2609453797340393, -1.1989039182662964, -1.6278579235076904, 2.070679187774658, -0.7218493819236755, 0.85993891954422, -1.0655945539474487, -0.18568329513072968, -1.0406010150909424, -0.42476314306259155, 0.4154769480228424, -0.31995338201522827, 0.4810043275356293, 0.8256292343139648, -1.644300937652588, 1.0038173198699951, 0.0925111249089241, 0.3338342607021332, -0.8217787742614746, 0.5655045509338379, -0.38652661442756653, 0.31920281052589417, -0.14511071145534515, -0.08384771645069122, 0.10823670774698257, 0.670909583568573, -0.7384783625602722, 0.5019673705101013, -0.10166995972394943, -0.26797252893447876, -0.9741054773330688, -0.6683739423751831, 1.9032578468322754, -0.5758640766143799, -0.5713655948638916, -1.1231975555419922, 0.8024492263793945, -0.3035558760166168, 0.9127227067947388, 0.7855374813079834, -1.8627163171768188, 1.8804060220718384, -0.771412193775177, -0.7329438328742981, 0.2997995913028717, -0.7090579867362976, -0.5706571936607361, -0.9617298245429993, -1.0011136531829834, 0.9500221014022827, -0.5841365456581116, 0.2944156527519226, -0.6538920998573303, 0.06960860639810562, -0.6063957810401917, -0.7441114187240601, 0.4813040792942047, 1.0462377071380615, -1.3174165487289429, 0.9096676707267761, 0.0008936012163758278, 0.4797937273979187, 0.2881912589073181, -1.4769386053085327, 0.3888286352157593, 0.5353546738624573, 0.49695372581481934, 0.6951083540916443, -0.35184505581855774, 0.1883656084537506, 0.20878732204437256, 0.4067421853542328, 1.156895637512207, -1.0164979696273804, 1.9412378072738647, 0.6869062185287476, 0.9622466564178467, -0.34245964884757996, -0.5756792426109314, -1.066658854484558, -0.0323946550488472, 0.16518251597881317, 0.013689301908016205, -0.3040390610694885, 0.6621749401092529, -1.0331677198410034, 1.121293544769287, -1.2184187173843384, 0.07333924621343613, -0.49060410261154175, -1.562011957168579, -0.4728454053401947, -0.13513967394828796, 0.9583438634872437, -0.2303369790315628, 1.251552700996399, 0.9333856701850891, 1.542272925376892, 1.2084864377975464, -1.3967328071594238, 0.7260504364967346, -0.08754625171422958, -0.23351633548736572, -0.7533081769943237, -0.553867757320404, -0.46203669905662537, -0.09546098858118057, 0.5480346083641052, -0.42337846755981445, 0.36714446544647217, 0.5964148044586182, -0.11905097961425781, 0.47356462478637695, 0.9441995024681091, 0.5724157691001892, -0.19228583574295044, 0.0967661589384079, -0.02742111124098301, -2.848541736602783, -0.333787739276886, -0.03304649144411087, 0.0888482853770256, 0.05183377116918564, 1.1415066719055176], [-1.0075260400772095, 0.11853627115488052, 0.3291098475456238, -1.263472318649292, 0.02622750960290432, -0.09738040715456009, 0.9390558004379272, -1.5244479179382324, 0.7585954070091248, 0.12310250103473663, 0.1759396195411682, 0.7520075440406799, 0.4008546471595764, -1.8027551174163818, -1.8032244443893433, -1.9914144277572632, -0.4693301022052765, -1.5902413129806519, 0.3361207842826843, -0.11765773594379425, -0.2806428074836731, 0.6007952094078064, -1.2894576787948608, -0.08880940079689026, 1.1354622840881348, 1.431052565574646, 0.5062008500099182, -1.0080679655075073, -0.36466991901397705, -0.9044856429100037, 0.4912886321544647, -0.05325096473097801, -0.530570387840271, -0.097394660115242, -2.283747911453247, 1.5113884210586548, 0.1639435589313507, -0.9998903870582581, -1.3077218532562256, -0.8851490616798401, 1.7990758419036865, -0.5302613377571106, 0.3046240508556366, -0.9790553450584412, 1.7130706310272217, 1.0683693885803223, -0.7883510589599609, 1.7620892524719238, 1.2770808935165405, 0.28122884035110474, -1.7997578382492065, 0.23469336330890656, -2.290583848953247, 1.0804874897003174, 0.8977952599525452, -0.2171640247106552, 1.4861176013946533, 0.1859772503376007, -1.3925504684448242, -0.35831204056739807, -0.21839489042758942, 1.7005316019058228, -0.040482018142938614, -1.3815656900405884, 1.0518256425857544, 2.0229501724243164, -0.40795692801475525, 0.22101259231567383, -1.1766482591629028, -0.39864879846572876, 0.97202467918396, -0.10249101370573044, -2.213001251220703, -1.2982012033462524, 0.6940838694572449, -1.262450098991394, -0.6817672848701477, 0.13080739974975586, 1.0116599798202515, -0.6185317039489746, -0.8066197037696838, -0.09277927130460739, -0.8151479959487915, 0.009637989103794098, -0.03748730570077896, -0.15127059817314148, 0.31208837032318115, -0.004561166279017925, 0.6470738649368286, -0.817141592502594, 0.38170379400253296, -0.044813912361860275, 0.7181401252746582, -0.026812344789505005, 1.143833041191101, 0.21886427700519562, -1.505145788192749, -2.368178129196167, 0.6750055551528931, -0.08876167982816696, -0.31615352630615234, 0.6401785612106323, 1.48623788356781, 0.974536120891571, -0.22602617740631104, 0.18631786108016968, -2.1355907917022705, -0.009869401343166828, 3.1724627017974854, 1.6390376091003418, -2.4896535873413086, -0.008050450123846531, 1.595649003982544, 0.10646692663431168, -0.23722407221794128, -1.5502382516860962, -0.7878327965736389, 1.154198169708252, 0.2452869564294815, -1.212295413017273, 0.8457715511322021, -1.711809754371643, -1.7348428964614868, 1.257930040359497, -0.2515696585178375, -0.4821399748325348, -0.8300464749336243, -2.6244871616363525], [0.9495531320571899, 1.3691647052764893, 1.3327677249908447, 0.4461807608604431, 1.7014294862747192, -0.0628456324338913, -0.9192131161689758, 1.2958757877349854, 0.3075587749481201, 0.24356257915496826, -0.9288844466209412, -0.6802555918693542, 2.1090872287750244, -0.20406755805015564, 1.5091999769210815, -0.7030417919158936, -0.9501240849494934, -0.8932791352272034, 1.4302964210510254, -0.9672317504882812, 2.0821125507354736, 0.20417700707912445, -0.3623010218143463, -1.8338490724563599, 0.690024733543396, -0.21774981915950775, 0.4139285087585449, -0.3145674765110016, -0.3922072649002075, 1.1726162433624268, 0.7118504047393799, -0.44553956389427185, 1.0817553997039795, -0.6789945960044861, 1.4163655042648315, 0.5067110061645508, -2.119119644165039, 0.17929264903068542, -0.014302127994596958, -0.9152926802635193, -0.31961193680763245, 1.1266748905181885, -0.7275444865226746, -1.5103323459625244, -0.5838260054588318, -0.17270196974277496, 0.8853003978729248, 0.20768079161643982, 0.18605473637580872, 0.9948927760124207, 0.05434775352478027, -0.1427846997976303, 1.032927393913269, -0.5009063482284546, 0.7160068154335022, -0.8769205808639526, 1.4474605321884155, 1.4578319787979126, 0.6081525683403015, 1.6338149309158325, 1.5654348134994507, -0.6436653733253479, 1.6495743989944458, -0.026735054329037666, -0.4736034870147705, 0.9666875600814819, -0.6879278421401978, -0.34887439012527466, -0.6953351497650146, -0.23353461921215057, 0.913829505443573, -1.0193573236465454, -1.622448444366455, 0.15661759674549103, 1.0676137208938599, -1.1270171403884888, -0.1624022275209427, 0.3491376042366028, -1.872999668121338, 0.1913483887910843, -1.151841402053833, 0.00693527702242136, -0.337911456823349, 1.0839842557907104, 0.29346713423728943, -1.466598391532898, 0.9565587043762207, 1.2808729410171509, -0.5095275044441223, -0.052242714911699295, -0.15327239036560059, 0.47860097885131836, 1.5785466432571411, -0.030062586069107056, -0.16546925902366638, -0.3037468194961548, 0.8664003014564514, 0.936529815196991, 1.2582569122314453, 1.0270395278930664, -0.009777755476534367, 0.2160218209028244, 0.7157925963401794, 0.4177480638027191, -2.1995112895965576, -0.4193161725997925, 0.8474350571632385, 1.7223058938980103, 2.0151689052581787, -0.27245497703552246, 0.6649518013000488, 0.5050366520881653, -0.7890710830688477, -1.7421915531158447, 1.2439014911651611, 0.4457254707813263, -1.1294232606887817, -0.31692183017730713, -0.35939598083496094, -0.262626588344574, 0.8547948002815247, -0.22884730994701385, -0.4576524794101715, -0.47427302598953247, 1.7238996028900146, 2.1804137229919434, -2.3392815589904785, -1.042074203491211], [0.23960675299167633, -0.6152375340461731, -0.9664722084999084, -1.6654274463653564, -0.445173054933548, -0.5424310564994812, 0.34659695625305176, 0.4873717725276947, 2.6248674392700195, 2.158332109451294, -1.5015279054641724, -0.4308764934539795, -0.18961788713932037, -0.39295506477355957, 0.8017473816871643, 0.21562747657299042, 0.17920060455799103, -1.033545732498169, 1.9675424098968506, 0.6530972123146057, -0.23074205219745636, -0.9113009572029114, -1.6163334846496582, 1.4177857637405396, 0.29681578278541565, 0.5683087706565857, -0.7766745686531067, -1.6814483404159546, -0.9605131149291992, 0.9326037764549255, -1.1230504512786865, 0.17294563353061676, -1.088677167892456, 1.450402855873108, -0.1733570396900177, -1.1915826797485352, -1.1067683696746826, 0.10057278722524643, 0.16335327923297882, 0.09772155433893204, -1.5721652507781982, -1.8347702026367188, -0.04925217479467392, -0.36305689811706543, 1.8719062805175781, 0.22387075424194336, 0.6578496098518372, -1.769918441772461, 2.3109655380249023, 0.7516200542449951, -2.4807379245758057, -0.2972213923931122, -3.0475986003875732, 2.3431406021118164, -0.08922535926103592, -0.10203792154788971, -0.2574179768562317, 0.6231669783592224, 0.42940381169319153, -0.40458789467811584, 0.613111138343811, -1.0489206314086914, -0.6361834406852722, -0.10255461186170578, 2.112416982650757, 1.2511454820632935, -0.3656976521015167, 0.2826787233352661, -1.0376427173614502, -0.01484991516917944, 0.8450416922569275, 1.8320727348327637, 0.11643943935632706, 0.9612916707992554, -0.44257405400276184, 0.2147628664970398, 1.4893290996551514, -0.7745238542556763, 1.3600794076919556, 0.1531280130147934, 0.11863067746162415, -0.060866642743349075, 1.6252241134643555, 0.8790664672851562, 0.14699746668338776, 2.1101596355438232, -0.7334734797477722, -0.45692503452301025, -0.018842890858650208, 0.3663935661315918, 0.3031974136829376, 0.38493672013282776, -0.6802968978881836, 0.3404446840286255, -0.7750133275985718, 1.6208093166351318, 0.13605985045433044, 1.4064881801605225, 2.11360239982605, -0.17686399817466736, -0.43124186992645264, 0.5971010327339172, -1.007601261138916, 0.7041069269180298, -1.1406642198562622, 0.5973278880119324, 0.11482415348291397, 0.263935923576355, 1.5213440656661987, -1.9695407152175903, -1.621958613395691, 0.8867729902267456, 0.0035071736201643944, -0.593214750289917, 0.5965838432312012, 0.7282432317733765, -1.4243751764297485, -1.1124308109283447, -2.1553635597229004, -0.9880403280258179, 1.8883720636367798, 0.5148816108703613, -0.8105604648590088, -0.4111218750476837, -3.047274351119995, 0.8228339552879333, -0.7010340690612793, 2.573596954345703], [-0.44162020087242126, 0.952438473701477, 1.2222639322280884, -0.007629367522895336, 0.08407045900821686, 0.8380585312843323, 0.6524472236633301, 1.477612018585205, -0.08048328012228012, -0.16016481816768646, -0.022177569568157196, 1.740366816520691, 0.614599883556366, 1.9921320676803589, -0.7400598526000977, -0.24654649198055267, 1.2773743867874146, -1.9955464601516724, -0.49276021122932434, 0.7402598857879639, -0.6055814623832703, 0.33142682909965515, 0.7124477028846741, 0.413863867521286, -0.6174044609069824, -0.9340152740478516, 0.9303035736083984, 1.179067850112915, -1.207964539527893, 0.4261133074760437, -0.277815043926239, -1.0260672569274902, -0.282108336687088, -1.6619081497192383, -0.23519361019134521, 0.7157169580459595, -0.11968476325273514, -0.9778620004653931, -0.6788640022277832, 0.6817455887794495, -1.429071307182312, -0.8167498707771301, -2.3248960971832275, 0.9709397554397583, 0.26986265182495117, 0.10961484163999557, 0.7727377414703369, -0.33129093050956726, 1.851515531539917, 1.9440597295761108, 0.2726268172264099, -0.6387683153152466, 0.8868075609207153, -0.5010702610015869, -0.41086122393608093, -0.8423647284507751, -0.05351792648434639, 1.1709141731262207, -0.44659316539764404, -1.1272857189178467, -2.288489580154419, 0.3852580785751343, 1.1783498525619507, -0.17867626249790192, 0.5367128252983093, -0.20524467527866364, 0.37550824880599976, 0.5844918489456177, -0.9795295596122742, 1.232434630393982, 1.1606992483139038, -1.4122623205184937, 0.9419257640838623, 1.6440201997756958, -1.850132703781128, -0.7002547383308411, 0.058661751449108124, -2.9766054153442383, 1.257513165473938, -0.9734837412834167, -0.17259758710861206, 0.16242007911205292, 0.7771929502487183, 1.3373697996139526, 1.035765290260315, -1.3022842407226562, -0.212471604347229, 0.9056376218795776, -1.1412200927734375, 0.03526308014988899, 1.825103998184204, 0.6447955369949341, 0.7350517511367798, 0.17599502205848694, -0.5846333503723145, 0.5146940350532532, -0.9596915245056152, 1.0696436166763306, 1.085042119026184, 1.0063755512237549, -0.6418564319610596, 0.9119061231613159, -0.48101094365119934, 0.12531805038452148, -1.1103777885437012, -0.7421097159385681, -0.1545930653810501, -0.15296481549739838, -1.0733473300933838, 2.1558029651641846, -0.4841301143169403, -0.5450036525726318, 1.7312716245651245, -0.42816153168678284, 1.2538824081420898, -0.11693640798330307, 1.5005234479904175, -0.32424792647361755, 0.17991819977760315, -0.5709139108657837, 0.006142234895378351, 0.3304730951786041, -1.0129863023757935, 1.5401840209960938, 0.594457745552063, -0.0029624979943037033, 0.5405033826828003, -0.359497994184494], [-0.9250475764274597, 1.5803159475326538, 0.11538193374872208, -0.9227914810180664, -0.9523919820785522, 0.880821704864502, -0.10220129042863846, -0.6236928105354309, 1.3115086555480957, 0.03219499811530113, 0.6351562738418579, 0.4411297142505646, 1.3544466495513916, 0.986649215221405, 0.07899434119462967, -1.183984398841858, -0.7990521192550659, -1.3738183975219727, -0.7298614978790283, 0.3168398439884186, 0.8314837217330933, -0.6248770952224731, -2.1506905555725098, -0.3938571512699127, -0.47350001335144043, -1.0601856708526611, -0.36732515692710876, -1.0074677467346191, 0.20375420153141022, 0.03282282128930092, -0.5289581418037415, -2.0009264945983887, -0.8150928616523743, 1.0634022951126099, 0.006949800066649914, -0.04587843269109726, 1.4715955257415771, -0.4162580370903015, 0.29940065741539, 0.39574047923088074, -1.2600966691970825, -0.17852455377578735, -0.06143319979310036, -0.2834300696849823, 0.058976370841264725, -1.3878264427185059, -0.05506980046629906, -0.11022411286830902, 1.4450525045394897, 0.8959226608276367, 0.5184774994850159, -1.037548303604126, -2.4052975177764893, 0.9799063801765442, -0.0967104434967041, 0.5017842054367065, 0.15062089264392853, -1.1540677547454834, -0.36395159363746643, 0.15721337497234344, -0.06050986424088478, 1.5543104410171509, 0.3299991488456726, -1.9274100065231323, -0.547058641910553, -1.3835244178771973, 1.1503225564956665, 0.329965740442276, 0.765605092048645, -1.870292067527771, 0.21276451647281647, 0.9294165968894958, 0.598129153251648, 0.442905068397522, -1.1176894903182983, -0.705522358417511, 1.4442259073257446, 0.1772954761981964, -1.4833523035049438, -0.011262478306889534, -0.2560088634490967, 0.08770647644996643, -0.4407743215560913, 1.3100061416625977, 0.37039729952812195, 1.2271634340286255, -0.9470820426940918, 1.1172443628311157, -0.30599671602249146, -1.5484168529510498, 0.731652557849884, -1.023561716079712, 1.8593238592147827, 0.8678382039070129, 0.9554557204246521, -0.25681358575820923, 2.1129696369171143, -2.030477523803711, 0.2079962193965912, -0.7586932182312012, -0.14664076268672943, 1.1531250476837158, 0.06099426746368408, 0.05445941537618637, 0.717889666557312, 0.9559429883956909, -0.02044537477195263, 0.8077981472015381, -2.053905487060547, 1.1667569875717163, -0.001993929734453559, 1.58942449092865, 1.2294068336486816, 1.6623475551605225, 2.0859365463256836, 0.46028298139572144, 1.690248966217041, -0.31809377670288086, 0.6952238082885742, 1.1280591487884521, 1.2583779096603394, -0.01918487623333931, -0.4505355954170227, 1.0246491432189941, -0.4277462065219879, -0.1488601416349411, -0.22444775700569153, 1.7957566976547241], [-0.6158627271652222, 0.08213627338409424, -0.5248544216156006, 1.5445632934570312, -0.5292037129402161, -0.5358462333679199, 0.8313878774642944, -1.5085991621017456, 1.085752010345459, 0.7902988791465759, -0.33906134963035583, -1.9123982191085815, -1.2908353805541992, -0.15344232320785522, -0.5326337814331055, -1.102625846862793, 0.019773265346884727, 1.920215368270874, -0.8208668231964111, -0.8372436165809631, -0.5536622405052185, 0.20831996202468872, 1.241049885749817, -0.2479529231786728, 1.8141759634017944, 0.3603731393814087, -1.509505271911621, 0.2768888473510742, -0.6667258739471436, 1.8313673734664917, 0.5000348091125488, -0.018428664654493332, -0.5900256633758545, 0.16271741688251495, 1.3837788105010986, 1.4716839790344238, -0.9208126664161682, 0.04381995648145676, -1.7428393363952637, -0.38844552636146545, 0.8208174109458923, -0.6537169218063354, 1.124982476234436, 0.1401367038488388, -0.45780444145202637, -1.5472831726074219, -0.5854416489601135, -0.953715980052948, -0.7371312379837036, -0.6134194135665894, -0.4764387905597687, -2.1487371921539307, 0.10300252586603165, 0.09405779838562012, 0.3849848210811615, 1.163347601890564, 1.553330421447754, -0.7061554789543152, 0.4719133973121643, -1.9580093622207642, 0.31315872073173523, -0.6267673373222351, 1.2088806629180908, 1.4552274942398071, 0.5603217482566833, -0.11009531468153, 1.1992733478546143, -0.3187260329723358, -0.37145882844924927, 0.48701420426368713, 0.9652803540229797, -0.23950663208961487, 1.3168764114379883, -1.2575597763061523, 1.5994399785995483, 1.2669286727905273, 1.5228761434555054, -0.682232677936554, -0.5268694758415222, 0.9791333079338074, -0.4476451575756073, 0.8843319416046143, -0.8190527558326721, -0.558176577091217, -0.3342307507991791, -1.1464695930480957, 0.5249307751655579, 1.350592017173767, -0.2683373689651489, -1.1430294513702393, -0.48271241784095764, 0.22190816700458527, -0.4040111303329468, 1.1136711835861206, -1.0230532884597778, -0.43197521567344666, 0.5090075135231018, -0.006321679335087538, -0.9123170971870422, 2.2447569370269775, -0.13722611963748932, -0.7304970622062683, -0.07567824423313141, 0.3748568594455719, 1.4966996908187866, 0.5567682981491089, 0.14599673449993134, -0.034658655524253845, 0.032393161207437515, -0.8413712978363037, -0.19909116625785828, 0.27692148089408875, -0.7072840929031372, 0.5098819732666016, 0.36434122920036316, -0.8389787673950195, 0.9336262345314026, 1.4750877618789673, 1.9846745729446411, 1.5607373714447021, -0.5363287925720215, 0.44954803586006165, -1.4048079252243042, 0.46406805515289307, -0.8071666359901428, 1.4540479183197021, 0.004474736750125885, 0.01704087108373642], [-1.9154280424118042, -1.4867900609970093, 2.5604937076568604, -0.4764433205127716, -0.34543749690055847, -1.6736098527908325, 0.3846500515937805, 1.205112099647522, 0.94817715883255, 1.4255303144454956, 0.2681502401828766, -1.361242413520813, 0.06453206390142441, -0.5773674249649048, -0.014183645136654377, -0.09629493206739426, 2.6946752071380615, -0.43546295166015625, 0.2180798053741455, -1.4241571426391602, 0.5760035514831543, 1.2106070518493652, -0.46306437253952026, -2.023083209991455, -1.572216510772705, -1.6211028099060059, 0.6223114132881165, -0.0174131877720356, -1.9419958591461182, -0.4746829569339752, 1.373108148574829, 1.6823346614837646, 1.4212251901626587, 0.8912345767021179, -1.4671229124069214, 1.6617929935455322, -0.988439679145813, 0.011661427095532417, 0.5496955513954163, 0.9269899129867554, 2.57002592086792, -0.48461437225341797, -0.7036548852920532, -0.00333771831355989, 1.006098985671997, -0.6341367959976196, -2.056638479232788, -0.4441869556903839, -1.176256537437439, 0.33797696232795715, -1.0756251811981201, 0.5733353495597839, 0.25649601221084595, -0.0971844345331192, 0.22733981907367706, -0.1562730222940445, 0.2573085427284241, -0.3400571346282959, -0.8090924620628357, -1.281052589416504, -0.7363096475601196, 0.16771842539310455, 0.014554009772837162, 0.4115980863571167, 0.9561557173728943, 0.8342071175575256, 1.3493635654449463, 2.762047052383423, -1.8205119371414185, -0.6499192714691162, 0.05259086191654205, -1.2310051918029785, -0.4527374505996704, -0.5159175395965576, 2.6389284133911133, 0.8068347573280334, 0.26981931924819946, -0.6833754777908325, -0.03829265758395195, 0.32355132699012756, -1.6814316511154175, -0.9145324230194092, 1.2563471794128418, -1.030724287033081, -0.45916545391082764, 0.015552897937595844, 0.7294397354125977, -1.8007144927978516, 1.2397617101669312, -0.5250563025474548, -0.6371228098869324, 0.6372992992401123, 0.15242967009544373, -1.1605812311172485, 0.7620553970336914, 0.14939478039741516, -0.4459996521472931, -1.0344411134719849, -1.4472014904022217, 1.619306206703186, -0.9258502125740051, 0.7272728085517883, -0.01985090784728527, 0.7250662446022034, 1.3549222946166992, 1.374091386795044, 0.7247408032417297, -0.8291018605232239, 0.07413842529058456, 0.40205028653144836, 0.2887018918991089, -3.071220636367798, 0.49244287610054016, -1.7724145650863647, -1.781935453414917, -0.22697876393795013, -0.9087541699409485, -0.3783750534057617, 0.3027366101741791, -0.8793888688087463, 1.0694117546081543, -1.7407245635986328, 0.34074440598487854, -0.5494018793106079, 0.6156412363052368, -1.0926202535629272, 1.0699833631515503, 0.6498395204544067], [0.5261813998222351, 0.034540556371212006, 0.037567831575870514, 1.485257863998413, 0.3562881052494049, -0.1311141699552536, -0.873170018196106, -1.0625661611557007, 1.0895605087280273, 1.203061580657959, 0.2835710942745209, -1.3274692296981812, 0.2482917308807373, 1.2979899644851685, 0.21648110449314117, -0.9219633340835571, 0.35606783628463745, -1.4418532848358154, 0.7990110516548157, 0.15369105339050293, 0.05526120215654373, 1.404484748840332, 0.7458359599113464, -0.2544234097003937, 0.5720862746238708, 0.45055216550827026, 0.9786557555198669, -2.2869176864624023, -0.09464766085147858, 0.8817567229270935, 0.4542013108730316, -1.6453545093536377, 1.0883617401123047, -0.027919145300984383, -0.4874114692211151, 1.2470307350158691, 0.6723790764808655, 1.2418550252914429, 0.43044304847717285, 0.6009923815727234, 1.1497458219528198, -0.12591741979122162, -1.1790950298309326, 2.1409926414489746, 0.012126343324780464, -0.49056339263916016, -0.007167846895754337, 1.765813946723938, -2.0561418533325195, -0.2378867268562317, -0.9132682085037231, 0.8481668829917908, 2.1884255409240723, 0.4568788707256317, -1.5560282468795776, -1.4660594463348389, -1.8342278003692627, 0.7147964835166931, -1.0747605562210083, 0.20980960130691528, 0.4578796923160553, 0.6977953314781189, 0.9982296228408813, -1.1162776947021484, -1.60544753074646, 0.5941896438598633, 2.1166110038757324, 1.3174376487731934, -1.1437122821807861, -1.237746238708496, -0.5539114475250244, 0.28326845169067383, -1.1083965301513672, 0.7902759313583374, -1.1331008672714233, -0.616969883441925, -1.1606277227401733, -0.6286569833755493, 0.170012429356575, 0.2558774948120117, 0.9920676350593567, -1.6073580980300903, 1.0460312366485596, 0.3347693681716919, -0.17954318225383759, 0.09439247846603394, -0.659669816493988, 0.7190518379211426, -0.8705541491508484, 1.2833023071289062, -0.7097665071487427, -1.645776629447937, -0.37236639857292175, -0.20069104433059692, 0.09503058344125748, 1.8404529094696045, 0.07410148531198502, 0.9793491363525391, -0.46739065647125244, 0.43456757068634033, 2.6837425231933594, -0.2362535148859024, -1.0462284088134766, 0.8046846389770508, 1.9124826192855835, -1.648646593093872, -0.25135377049446106, -1.2625099420547485, 0.5094383358955383, -0.708020806312561, 0.6193879842758179, 0.026201436296105385, 0.7715908288955688, -0.10857532173395157, -1.6365938186645508, 0.48003703355789185, 0.4515257179737091, 1.7426255941390991, -1.7691709995269775, -1.7318332195281982, -0.8922529816627502, -0.17222340404987335, 0.09498261660337448, -0.889493465423584, -0.39613744616508484, 0.5394115447998047, -1.0561007261276245, 0.10896492004394531], [0.08848920464515686, 0.37930068373680115, -0.303292453289032, -0.0609104298055172, 0.6882441639900208, 0.4733735918998718, 0.7450793385505676, 1.5536372661590576, -1.1527371406555176, 2.17441987991333, 0.535245418548584, -0.9302583932876587, 0.6373268365859985, 0.5102418661117554, -0.3946390151977539, 1.1470626592636108, 2.1483957767486572, 1.699509859085083, -1.3705333471298218, -0.6569100022315979, -1.0293614864349365, 0.5164161324501038, 0.25144749879837036, 0.5879125595092773, 0.7263184785842896, -0.4802209734916687, 0.3525764048099518, 0.054261963814496994, -2.106501817703247, -1.5743964910507202, 1.274857521057129, -0.9839341640472412, -0.03542608395218849, -0.9391441345214844, -1.6162234544754028, -0.30009937286376953, 1.4437508583068848, 1.1120816469192505, 2.0616002082824707, -0.8894633054733276, -0.13254320621490479, -0.49898216128349304, 0.7240588068962097, 0.7122242450714111, -0.08166521042585373, 0.3570009768009186, -0.5419843792915344, -1.225123405456543, 0.09548777341842651, -0.4383811950683594, -0.9086015224456787, 0.8840702176094055, 1.757115364074707, -1.8452824354171753, 0.6374784708023071, 2.2538764476776123, 0.9176099896430969, 2.0154595375061035, -0.1286558359861374, 1.4912362098693848, 1.3515982627868652, -0.6927669644355774, -0.3343104124069214, -0.705323338508606, 2.367678642272949, 1.549201488494873, -0.5160741806030273, 0.4723496437072754, -1.2994338274002075, -0.11279390007257462, -1.7907447814941406, 0.5876505970954895, -0.29561248421669006, -0.17489056289196014, -0.059161849319934845, -0.6347610354423523, 0.7419407963752747, -1.6353189945220947, 0.06840430200099945, -1.3892408609390259, -0.22136737406253815, -0.6399743556976318, 0.8865654468536377, 0.4895544648170471, -0.9915675520896912, 0.8798362612724304, -1.351435661315918, 0.7539495825767517, 1.168888807296753, 0.01218224037438631, -0.5391069650650024, -0.033084988594055176, 0.31325867772102356, -0.06494281440973282, 1.1132713556289673, -0.19508133828639984, -0.5022904872894287, 1.4079211950302124, 0.06786801666021347, -1.3912802934646606, -0.017179543152451515, -0.6845048069953918, 0.45781606435775757, -1.0264980792999268, 0.9420714974403381, -0.0037377835251390934, 1.4778521060943604, 0.27323436737060547, 0.675719141960144, -0.10409756004810333, -0.5252934694290161, 1.4576950073242188, -0.9015889167785645, -0.29392489790916443, 1.0449217557907104, 1.2800509929656982, 0.38817888498306274, 2.1230223178863525, 1.9321907758712769, -1.0320723056793213, -0.8696765899658203, -2.3913943767547607, -0.03773350268602371, 0.26285719871520996, -1.4012866020202637, 0.004064739681780338, 2.157578468322754, 0.09795279055833817], [-1.7202470302581787, 0.6565225720405579, -0.4923691749572754, 0.020061494782567024, -1.223546028137207, -0.003689279081299901, -0.23370607197284698, 0.5521968603134155, -0.26731210947036743, 0.3676080107688904, -1.4129377603530884, -1.507748007774353, 0.9805610775947571, -0.9099335670471191, -0.10934196412563324, 0.6599475741386414, -0.9203002452850342, -0.008328935131430626, -0.5541449189186096, -0.0232948437333107, 0.5345120429992676, -1.110609531402588, -0.8144350051879883, 0.8004288673400879, -1.3196028470993042, 0.4134089946746826, -1.6783291101455688, 0.05082755908370018, 0.6265084147453308, -1.0821266174316406, 0.6853083372116089, -1.0050551891326904, 0.8160513639450073, 0.28124597668647766, -1.2335491180419922, 1.8841928243637085, 0.37914782762527466, 4.164571762084961, -0.16821759939193726, 0.9804775714874268, 0.8572930097579956, -0.6350287199020386, -0.21887923777103424, 0.33494213223457336, -0.6302425861358643, 1.0013786554336548, 1.1528205871582031, 1.057813286781311, 0.3437533378601074, -0.2581164240837097, 0.8167206048965454, -0.24562132358551025, -1.0904390811920166, 0.43057355284690857, 0.23237109184265137, -0.667636513710022, 0.9768046140670776, -0.6932281255722046, -1.4081610441207886, -1.7176164388656616, 0.3739975094795227, -0.004981375765055418, -0.3554430902004242, 0.9513135552406311, -1.133936882019043, 0.8117672801017761, 0.32935425639152527, 1.4878003597259521, -1.0534437894821167, -0.4491927921772003, 0.011191566474735737, 1.4776208400726318, -0.46827948093414307, -1.99833083152771, -0.5822295546531677, -0.4100905656814575, -0.19070342183113098, -0.9392359852790833, 1.2642626762390137, 1.1246494054794312, 0.49196097254753113, -0.8056202530860901, -0.5019873380661011, 0.35161247849464417, 0.32403674721717834, 0.059678953140974045, -0.8955587148666382, -0.5742112994194031, -1.2289867401123047, -0.08743126690387726, 0.18873324990272522, -0.5479856729507446, 0.5186260938644409, -0.3987632691860199, 0.03674333915114403, 0.5345502495765686, -0.033188894391059875, 0.07001598924398422, -0.5940873026847839, 0.3799467086791992, 0.03813447058200836, -0.10499396920204163, -1.0680662393569946, -0.2791730761528015, 0.2978590726852417, 0.2180662453174591, -0.08487989753484726, 0.48896241188049316, 0.7894688248634338, -2.50215482711792, -1.0466387271881104, 0.8505341410636902, -1.110116958618164, -2.041841983795166, 0.5869250893592834, -1.246596097946167, -0.3182874321937561, 0.3557535707950592, 0.11388064920902252, 0.5854392647743225, -1.059535026550293, 1.7817953824996948, 1.1822603940963745, 1.78901207447052, -0.7178133726119995, 1.312781810760498, -1.91548752784729, -0.8705666065216064], [0.11068614572286606, 2.301638603210449, 0.8000109195709229, -0.44352442026138306, 0.5858115553855896, 1.4885684251785278, -0.3382718861103058, 0.7231788039207458, 0.41995084285736084, 0.17804457247257233, -0.03076324239373207, 0.6897040605545044, 0.34061357378959656, -0.7490102648735046, -0.38328200578689575, -0.3680680990219116, 0.8537282347679138, -2.5946834087371826, -0.07224888354539871, -1.0174856185913086, 0.9185577034950256, -0.2063307911157608, 2.2879130840301514, 0.9261802434921265, -0.3025818169116974, 1.1934525966644287, -0.2591366767883301, 0.2063944935798645, -0.8408004641532898, -0.35725387930870056, -1.5439081192016602, -0.9252646565437317, -0.6629365682601929, 1.01939857006073, -0.7684518098831177, -1.7083948850631714, 1.5569641590118408, -1.0795375108718872, -1.1038285493850708, 2.6261327266693115, 0.9146869778633118, -0.29389938712120056, -0.15828083455562592, 0.2894916534423828, 0.39372196793556213, -0.6741621494293213, -0.10345837473869324, 0.307563841342926, 1.0378451347351074, 1.1754902601242065, -0.5729215145111084, 0.9107866883277893, 0.7767149806022644, -0.3171465992927551, 2.0749309062957764, 1.6954163312911987, 1.0800354480743408, -0.4182994067668915, -0.00600137235596776, -0.6031744480133057, -0.4153141379356384, 1.2160207033157349, -0.5209140181541443, -0.06425425410270691, 0.2445671707391739, -1.1659715175628662, 2.3931186199188232, -0.1806894838809967, 0.3160446286201477, 0.7020689845085144, 0.6630762219429016, 2.330122232437134, -0.3998297452926636, -0.10364892333745956, 0.01722296141088009, -1.490812063217163, 1.9068185091018677, 0.6076216101646423, -0.28187575936317444, -0.6143874526023865, 1.642767071723938, -1.1113067865371704, 0.09652268886566162, 0.8523145914077759, 0.5495249032974243, 0.10916599631309509, 1.8534176349639893, 0.36664456129074097, -1.5322779417037964, 1.612144112586975, 0.04232889786362648, 0.1731029897928238, 0.1711825579404831, 1.925477147102356, 1.4735431671142578, -0.27755045890808105, -1.2594908475875854, -1.8022817373275757, -0.14053010940551758, -0.5974140167236328, 1.1136064529418945, 1.3392994403839111, 0.5052404403686523, 1.2037378549575806, -1.8395962715148926, -0.45087990164756775, 0.4111126661300659, 0.4958760738372803, 1.1491899490356445, 0.4526143968105316, -1.203431248664856, -0.021506819874048233, 1.1507972478866577, 0.02870236337184906, 0.6459420919418335, -1.2463473081588745, -1.1693717241287231, -0.43919816613197327, -0.7455887198448181, -0.7713250517845154, 1.48738694190979, -0.14021660387516022, 0.46745336055755615, 0.04145461320877075, -1.1160013675689697, 0.9290597438812256, 0.4415043592453003, 1.3418036699295044], [-1.0632296800613403, -0.9518004655838013, 0.2508580684661865, -0.8154480457305908, 0.2674594223499298, 0.2355409860610962, -0.24510037899017334, 0.03295554593205452, 0.4733724296092987, -2.2348082065582275, 1.421452283859253, -0.4939000904560089, 0.014676036313176155, -0.1392541527748108, 0.6385570764541626, 0.7139182686805725, -0.9612033367156982, 0.054017841815948486, 1.9601635932922363, -1.541282057762146, 0.49493908882141113, 1.2869858741760254, 0.5204080939292908, -0.817321240901947, 0.9976617097854614, 0.21552139520645142, 0.49238336086273193, 0.3745162785053253, -0.875474214553833, -1.0129886865615845, 0.9947607517242432, -0.07884581387042999, 1.197432518005371, 1.1746584177017212, -0.5309704542160034, 0.9897462725639343, -0.7655070424079895, 0.5649265050888062, -0.2351846992969513, 0.10105999559164047, -1.8296611309051514, -0.340378999710083, -1.3517218828201294, 0.7603275179862976, -0.24875086545944214, -0.45682579278945923, 0.6976255774497986, -1.3163856267929077, -0.1628498136997223, 1.388291597366333, 1.4289708137512207, 0.05894593894481659, -0.05898712947964668, 0.20934031903743744, -0.5783843994140625, 1.7709599733352661, 0.36272066831588745, -0.21565453708171844, -0.7342293858528137, 1.7858439683914185, -0.7813736796379089, 0.04341680556535721, 0.9822879433631897, -0.08571161329746246, -0.5399422645568848, -0.5093246698379517, -0.16166472434997559, -2.0562243461608887, -0.5307754278182983, -1.4138864278793335, 1.7402114868164062, 0.947860598564148, -0.2416798323392868, 0.46792447566986084, -1.5202569961547852, 0.7122060060501099, 1.0278761386871338, 0.11929920315742493, 0.9244219064712524, 1.7503753900527954, -1.1731226444244385, -0.614656388759613, -0.682140588760376, 0.07738985121250153, 0.5638154745101929, -0.6608991622924805, 1.0934069156646729, -0.3881983458995819, -0.17703713476657867, 0.45743700861930847, -0.5451791286468506, -0.3465569019317627, -0.29507288336753845, -0.8436191082000732, -0.23931194841861725, -0.09385974705219269, -0.4532855451107025, -1.1963976621627808, 1.365646481513977, -0.012005474418401718, 1.4182546138763428, 0.62876957654953, -1.106654405593872, 0.747087299823761, 1.5618526935577393, -0.9154189825057983, 2.271074056625366, 0.2056931108236313, 0.3166356682777405, 0.48534268140792847, -0.5491438508033752, 0.23860739171504974, 1.1037739515304565, -0.5214847326278687, -0.2230728417634964, 0.914897084236145, -1.6179934740066528, -0.0030691404826939106, -1.867443561553955, 0.0429094024002552, 0.2741139531135559, 1.2448949813842773, 0.5420441627502441, -0.5212832093238831, 0.5732113718986511, 1.5289289951324463, 1.221048355102539, -1.118248701095581], [1.3513946533203125, -0.4010506272315979, -1.8127033710479736, -1.063232183456421, 1.115753173828125, 0.40831300616264343, -0.057170696556568146, 0.095310278236866, -0.06694603711366653, 0.9054879546165466, -0.14423160254955292, -0.6872647404670715, -0.5238622426986694, -0.04004353657364845, 0.010555674321949482, -0.12783443927764893, -2.1847424507141113, 0.4889805316925049, -0.9924360513687134, -1.2379121780395508, 0.7746155261993408, 1.1187777519226074, -1.1034811735153198, 0.2767578065395355, -0.7990652918815613, -1.1446893215179443, -0.21905963122844696, -1.338651180267334, 2.287782669067383, -0.7271995544433594, -0.4459283649921417, 1.649180293083191, -0.8157678246498108, 1.4696626663208008, -0.27349767088890076, 0.7877621650695801, -0.20269054174423218, 1.1736063957214355, -2.4526658058166504, -1.8983174562454224, 0.5754233598709106, 0.7423513531684875, 0.2702120542526245, -1.4272947311401367, -2.1549956798553467, 0.7464961409568787, 0.5870116353034973, -1.5193246603012085, -0.35502204298973083, -0.6281874775886536, 0.1134805977344513, -0.26656022667884827, 0.3589807152748108, 1.0840774774551392, 0.6337548494338989, 0.6976792216300964, 0.3892231285572052, -0.550303041934967, -1.1857383251190186, 0.8221402764320374, 0.13727974891662598, 1.2328242063522339, 0.07962944358587265, -0.2872527241706848, -0.2668415307998657, 0.20925311744213104, 0.7315426468849182, 0.22033244371414185, -1.2440282106399536, 0.7541980743408203, 1.4383392333984375, -0.9076144099235535, 0.40599432587623596, 0.1924566775560379, 1.3955212831497192, -1.0575213432312012, -1.1908295154571533, -1.1799968481063843, 0.39493533968925476, -0.7736594080924988, 0.3911643624305725, -1.3991445302963257, 0.013135538436472416, 0.5068346858024597, 1.2820441722869873, 1.3884936571121216, -1.2010570764541626, -1.8108584880828857, 0.5727859139442444, 0.1603269726037979, -0.7794339060783386, -1.105899691581726, -0.37093549966812134, -0.02665577083826065, -0.23241852223873138, -0.7252698540687561, -0.7228087186813354, -0.7156789302825928, 0.09234879910945892, -0.28371384739875793, -0.3635920584201813, 0.6891435384750366, -0.6097677946090698, -1.1786011457443237, -0.15384148061275482, 1.4065203666687012, -0.2975209951400757, 0.8097922801971436, -0.8158605098724365, 0.32206296920776367, 0.8072841167449951, 0.42542171478271484, 0.7348028421401978, 0.3776165246963501, 0.7107653021812439, 2.3883702754974365, 0.6690073609352112, -1.7175418138504028, -0.33982646465301514, -0.8754248023033142, 0.5957417488098145, -1.166879653930664, 0.09458966553211212, -0.5504216551780701, -0.6106564998626709, -0.9602755308151245, -0.026817727833986282, 0.3002031445503235], [0.08113507926464081, -1.2665088176727295, -0.38007351756095886, 1.403639793395996, -0.23528271913528442, 0.43970149755477905, 0.6474074721336365, -2.0817856788635254, -2.6291263103485107, 1.1084092855453491, 0.6008807420730591, -0.016829093918204308, 0.12841898202896118, 1.7154730558395386, 0.2776632308959961, -0.7238329648971558, -0.01838293857872486, 1.7114713191986084, 2.702859878540039, 0.21830131113529205, -0.5168461799621582, 0.02158472314476967, 1.1192331314086914, 0.5496529340744019, -0.9820302724838257, 2.254892349243164, -1.1668791770935059, 1.5522258281707764, -0.9034914374351501, -0.09226008504629135, 0.1717681735754013, 1.5842268466949463, -0.24503503739833832, 0.7009538412094116, 0.543490469455719, 0.704024076461792, 0.41080206632614136, -1.0086885690689087, -1.4819650650024414, -0.13019034266471863, 0.05035964027047157, -0.3511718809604645, 0.17649920284748077, 0.6516426205635071, 0.432807594537735, 0.20519116520881653, -0.046390749514102936, 0.2970466911792755, -1.2385715246200562, 0.03330395370721817, -0.717847466468811, -0.18176861107349396, -1.3153234720230103, 1.5572130680084229, -0.1277778148651123, 0.7996224761009216, -0.6181573271751404, -1.4143304824829102, 0.0772302895784378, 0.7759631872177124, -0.5109527111053467, -1.788025140762329, 0.6218525171279907, 0.36005279421806335, -0.43360090255737305, 0.48048990964889526, 0.10275045782327652, -2.0225446224212646, -0.010765613988041878, 0.4181451201438904, 0.5163038372993469, 0.06237312778830528, 1.0996787548065186, -1.9819306135177612, 0.8338620066642761, 0.2785916328430176, 1.2374969720840454, 1.0877792835235596, -1.0123724937438965, -0.21266469359397888, -1.3038456439971924, -0.13673901557922363, 2.2593281269073486, 1.5742225646972656, -0.5121874809265137, -0.3265278935432434, 1.5285305976867676, -1.6697148084640503, -0.07831816375255585, -0.5069795846939087, -0.3311627507209778, 1.6633621454238892, 0.8951905965805054, 0.6639277935028076, -1.5624101161956787, 1.2253228425979614, -1.3008065223693848, 2.2232470512390137, -0.4967209994792938, -0.02275100164115429, -0.14260943233966827, 0.08814682811498642, -0.6367435455322266, 0.5442110300064087, 0.9115024209022522, -0.8788326978683472, -0.22176700830459595, 0.9068222045898438, 0.7820969223976135, -1.9403094053268433, 0.9962109327316284, 0.46165913343429565, -0.2271798700094223, -0.6440523266792297, -1.8169506788253784, 1.7052470445632935, -1.1179569959640503, 1.4482612609863281, 0.14530183374881744, 0.47348901629447937, -0.5424546599388123, -0.9754688739776611, 0.6045718193054199, -1.0189508199691772, -0.29974421858787537, 1.0918023586273193, 0.5703961849212646, -1.598214864730835], [1.0916308164596558, -1.6381679773330688, -0.169958233833313, 0.6869763135910034, -0.7026675939559937, 0.7116084694862366, -0.538727343082428, -0.3696196675300598, -0.3914017379283905, 1.0748519897460938, -0.6224707961082458, -0.5082393884658813, 0.021772701293230057, 2.332869052886963, 2.26131272315979, -0.09589886665344238, 1.3841142654418945, 0.17838843166828156, -0.2230513095855713, 1.8143359422683716, -0.23586414754390717, -2.113065719604492, -0.2165055274963379, -0.3446047008037567, -0.04078386723995209, -0.5981741547584534, 0.12500275671482086, -0.029762614518404007, 0.775300145149231, 1.197148323059082, -1.349913239479065, 0.6169272661209106, -0.8831488490104675, -0.8173583149909973, 1.1340371370315552, -0.1409696787595749, -0.9345551133155823, -0.0011723022907972336, 0.22897139191627502, -1.329994797706604, 0.10877598077058792, -0.2687074840068817, 1.1473041772842407, 0.43496620655059814, 0.6962064504623413, 0.9987988471984863, -0.28161191940307617, 2.174616813659668, 2.2428996562957764, -0.21643735468387604, 0.6613761186599731, 0.9177119135856628, 1.2751102447509766, -1.1249545812606812, -0.3235982358455658, -0.09447291493415833, 0.664076030254364, 0.7480437755584717, -0.7796148061752319, 0.7532975673675537, -0.48086991906166077, -0.8079856038093567, -1.5935609340667725, -1.4279084205627441, 1.4698244333267212, 1.3489453792572021, -0.10742652416229248, -1.3130922317504883, -1.0493676662445068, -0.8012628555297852, 1.03914213180542, 1.9633808135986328, -1.3901764154434204, 1.696142554283142, -1.9932624101638794, -0.5109761953353882, -0.06532174348831177, 1.659759759902954, 0.8389346599578857, 1.5768682956695557, -1.2958223819732666, 0.2176954746246338, 2.2278006076812744, 0.8841326236724854, 2.22240948677063, -0.9270198941230774, 0.1565299928188324, 0.42642566561698914, -1.2650933265686035, 1.6537591218948364, -0.010908938944339752, 0.2082534283399582, -1.5071784257888794, -0.25663936138153076, 1.0145759582519531, 0.7740088701248169, -1.2221877574920654, 0.8948034644126892, 0.4328605532646179, -0.010726094245910645, 0.17661726474761963, -0.419329434633255, -0.4636453092098236, 0.6250085234642029, -0.11520275473594666, -0.8871819972991943, -0.7037836313247681, 0.2456938475370407, 0.4042391777038574, -0.3466941714286804, 0.3024388551712036, 0.07709647715091705, -1.2012033462524414, 0.9970953464508057, -0.9358325004577637, 1.2427679300308228, 1.2671475410461426, -0.2945513129234314, -0.5362306833267212, -0.2819395959377289, 0.6663253903388977, -1.8457621335983276, 0.8052084445953369, -0.8203404545783997, 0.4977955222129822, -1.725554347038269, 0.15966685116291046, -0.1388917714357376], [-0.05039847642183304, -0.3143625259399414, -1.4973818063735962, 0.11274950951337814, 0.3628145158290863, -1.3545387983322144, -0.9856789112091064, -0.01683737151324749, 0.6154146194458008, -1.2457174062728882, -0.2605387568473816, -0.3244799077510834, 0.3024706244468689, -0.17065005004405975, 1.6843087673187256, -1.595523715019226, 0.7856277823448181, 0.7355853915214539, 0.2276192456483841, -1.0321918725967407, 0.727849006652832, -1.2883409261703491, 0.6398186087608337, 1.3265256881713867, 0.3853282928466797, -1.2217005491256714, -0.05121060088276863, 1.8150814771652222, -2.620974540710449, 0.06715047359466553, 0.4523511230945587, 1.3743969202041626, -1.0041635036468506, -0.6666487455368042, 1.1231366395950317, -1.805788278579712, 1.3374453783035278, -0.6839849352836609, -0.08977366983890533, -0.6888865232467651, -0.9923465847969055, -1.5453212261199951, 0.07273631542921066, 0.9639766216278076, 0.7094314694404602, -0.022748364135622978, -1.79636812210083, 1.969823956489563, 0.5653128027915955, -1.2959727048873901, 1.0348807573318481, 1.1505354642868042, 0.47555503249168396, 0.18311497569084167, -0.8059744238853455, 1.3224244117736816, 1.3795363903045654, 0.7332776188850403, 0.6102445125579834, -0.5546830892562866, 0.002854633843526244, 1.1082841157913208, -0.4905339181423187, -1.0797455310821533, 0.2849859893321991, 0.4182202219963074, -0.21261145174503326, 1.1290920972824097, -0.8377382755279541, -0.356872022151947, 0.47799935936927795, 0.9043005108833313, 0.8625947833061218, 1.946594476699829, -1.5187474489212036, -0.9735564589500427, -0.7697905898094177, 0.4530424177646637, 0.5605730414390564, -1.1364742517471313, 1.4188193082809448, 2.3597540855407715, 0.6468399167060852, 0.4114994406700134, 0.13556496798992157, -0.005411085672676563, 0.7578100562095642, -2.4528207778930664, 0.2034560889005661, -0.7757339477539062, -0.0729367583990097, 0.6306728720664978, 0.9477847218513489, -0.7511662244796753, -1.5901992321014404, 2.1973977088928223, -2.402527093887329, -1.4019207954406738, -0.43735846877098083, 0.8116924166679382, -1.6890507936477661, -0.22470708191394806, 0.07044456899166107, 1.3310790061950684, 0.5127508044242859, -1.611053228378296, -0.6400274634361267, 0.2335021197795868, 1.0026941299438477, 2.9752140045166016, 1.0211808681488037, 1.0296798944473267, 1.077511191368103, -1.173815369606018, -0.20435132086277008, -0.32730403542518616, 0.5980780720710754, 0.878990113735199, 0.19846555590629578, -0.7558797597885132, 0.9327235221862793, -0.679492712020874, 0.418962299823761, -0.4472767412662506, 1.1357619762420654, 0.9633265733718872, 0.6285474896430969, 0.5015641450881958], [1.0130776166915894, -0.7864014506340027, 0.5113205909729004, 1.389291763305664, -0.7707883715629578, -0.561184287071228, 0.1267482042312622, 1.8030036687850952, 1.4434617757797241, 0.6932404637336731, 1.0357518196105957, 0.23365771770477295, -2.4739491939544678, 0.8066311478614807, -1.428288459777832, 0.353059321641922, -0.5826242566108704, 0.12335943430662155, -0.3074113130569458, -0.5215949416160583, 0.9225099086761475, 1.724492073059082, 0.9102739691734314, 0.3337315022945404, 1.1886636018753052, 1.126879334449768, -1.4131865501403809, 0.3066501319408417, 1.4518427848815918, -1.7547643184661865, 0.8012366890907288, 1.4022866487503052, 0.022460052743554115, -1.2596111297607422, -1.1998603343963623, 1.0560745000839233, -0.20011547207832336, 0.1404639631509781, 0.8407453894615173, 0.26884210109710693, -0.2373456507921219, -0.9228209853172302, 1.0645990371704102, 0.18532906472682953, 0.01983194798231125, -1.465640664100647, 1.185648798942566, 0.032838840037584305, 0.8221725225448608, -0.7181583046913147, 2.0746355056762695, -1.3908690214157104, -0.6656644940376282, -0.5371475219726562, -0.9049251079559326, -0.2215675413608551, -1.5490542650222778, 0.7170756459236145, 0.41560110449790955, 0.9357171654701233, -0.00031023818883113563, -0.7868099808692932, -0.7753104567527771, -1.2495068311691284, -1.5949277877807617, -0.4845733940601349, -0.6498680114746094, 2.0858097076416016, 0.573018491268158, -1.7294713258743286, -0.6249144673347473, -0.2705182433128357, -0.526275098323822, 0.648319661617279, 1.1553822755813599, 0.3693240284919739, -1.1547964811325073, -0.019475262612104416, -0.26862311363220215, 0.7424633502960205, 0.9194284677505493, 0.5152742862701416, 0.0511002391576767, -0.5796212553977966, 1.7153724431991577, 1.6784393787384033, 0.3906487226486206, -0.07715217024087906, -1.219067096710205, -0.3239510953426361, 0.21840673685073853, -0.06106535345315933, 0.04790987819433212, 0.20991484820842743, -0.9219363331794739, -0.40526801347732544, -0.23353207111358643, -0.5438661575317383, -1.0416990518569946, 0.9533186554908752, -1.0006519556045532, 0.1631958931684494, 1.7788619995117188, 1.1876795291900635, -2.528144598007202, 1.3516124486923218, -0.29943227767944336, -0.4918513298034668, 0.1432773470878601, -0.23774003982543945, 1.1695481538772583, 0.7144498825073242, -0.7689449191093445, 0.9166256189346313, 1.3289536237716675, 0.2779086232185364, -1.1524094343185425, 1.0568881034851074, 0.75933438539505, 0.7558808922767639, 0.3179934620857239, 1.7211037874221802, 0.42881399393081665, -0.089866504073143, 0.13551335036754608, 1.0256143808364868, 0.8633447289466858, 1.1018683910369873], [0.4748218357563019, -0.8180170655250549, 0.5979294776916504, 0.9146180748939514, -0.6431229114532471, -0.168821319937706, 0.33931905031204224, -0.9168443083763123, 0.33068087697029114, -0.1960698962211609, -0.3872237205505371, 0.40290528535842896, 0.9002314805984497, 0.4675845205783844, 1.7673759460449219, -0.6529120206832886, -0.988325297832489, -0.3507428467273712, 0.8222880959510803, 1.0356122255325317, 1.7935235500335693, -1.060024619102478, 0.48283228278160095, -0.24154913425445557, -0.7426527738571167, 1.7612380981445312, -0.45662692189216614, 2.3955202102661133, 0.23757560551166534, -0.5226697325706482, -0.8093900084495544, 1.1210464239120483, 0.8475357890129089, -1.0341200828552246, 0.8122954964637756, 0.8686690926551819, -0.5146943926811218, -0.9869028329849243, 0.28732770681381226, 0.3174700438976288, 0.41740214824676514, 1.8365432024002075, 1.2015846967697144, 0.3750177025794983, -1.064389705657959, -0.6704902052879333, -0.4179832637310028, -0.15310660004615784, -1.299030065536499, 0.49727174639701843, 0.755536675453186, 0.11785750836133957, -1.4399547576904297, 0.05397282540798187, -1.1329169273376465, 0.9951728582382202, 1.4366739988327026, -0.33887749910354614, -0.3974493443965912, -0.8086950182914734, 1.2956280708312988, -0.9364774823188782, -0.08156483620405197, -0.298302561044693, 1.4658730030059814, 0.6841403841972351, 0.8081991076469421, 1.4454658031463623, 0.6897465586662292, 0.0421527735888958, -0.2614865303039551, -1.2173173427581787, 0.034591224044561386, -0.7963642477989197, -0.6973084211349487, -1.016822338104248, -0.9049167037010193, 0.34689369797706604, 0.25200095772743225, 1.7426091432571411, 1.0333919525146484, -0.09239412099123001, -0.15976227819919586, -1.2510206699371338, 1.1721718311309814, -0.04225703701376915, 0.8157657980918884, 0.18546494841575623, 0.8013362288475037, 1.1562755107879639, -0.8389776349067688, 0.4539191424846649, -1.923758625984192, 0.8500393629074097, -0.21949301660060883, -0.25238773226737976, -1.2788810729980469, 0.03677183762192726, 0.9101310968399048, 0.968345046043396, -2.332314968109131, 1.4737366437911987, -0.17554248869419098, 0.1480211317539215, -0.11878664046525955, -0.997326672077179, -0.7154873013496399, 0.050879500806331635, -0.9209209680557251, 0.8688066601753235, 2.1678860187530518, 1.4799433946609497, 0.8779578804969788, 1.0175641775131226, 1.0862327814102173, 0.21445614099502563, 1.1025618314743042, -0.04827028512954712, -0.7947803139686584, 1.2711788415908813, 0.20445001125335693, -1.7257035970687866, -0.21604380011558533, 1.19480562210083, 1.1608319282531738, 0.4979967772960663, 0.9153872728347778, -0.3081999719142914], [-0.27306050062179565, -2.559305429458618, 0.6971169710159302, 1.432759165763855, -0.8880359530448914, 1.8030766248703003, -1.346561074256897, -1.455372929573059, 0.7672550678253174, -0.4516466557979584, -0.011441764421761036, 0.4570502042770386, 0.061487894505262375, -0.5523189902305603, 0.17562462389469147, 0.6012321710586548, 0.7844652533531189, 0.478372186422348, -0.7064924240112305, -1.6200745105743408, -1.2783052921295166, 1.0759670734405518, 0.7081963419914246, 1.0004229545593262, 0.016963375732302666, -0.10074379295110703, -1.2423090934753418, 0.6182340383529663, -0.11281854659318924, -1.2984507083892822, -0.5592150092124939, -0.056987933814525604, -1.6423677206039429, -1.8327646255493164, 0.04800441116094589, 0.32715728878974915, -0.24974779784679413, -0.1605021059513092, 0.14502201974391937, 2.1025984287261963, 0.45362022519111633, -0.7017590403556824, -1.495262861251831, -0.1382075697183609, 0.12193349003791809, 1.1802492141723633, -0.7354224324226379, 0.41786032915115356, 1.402494192123413, 0.22753137350082397, -0.19004619121551514, -0.5021227598190308, 0.5309950709342957, 0.9900112748146057, 0.28954702615737915, 0.8701871633529663, 0.1389520913362503, -1.9429477453231812, -1.1354435682296753, -0.9450462460517883, 0.21625031530857086, 0.523011326789856, 1.495356798171997, 0.3167143166065216, -1.5552459955215454, 0.37152963876724243, 0.5835583806037903, -1.1640328168869019, 0.12880055606365204, 1.34133780002594, -1.4846857786178589, 0.6313617825508118, -0.4361873269081116, -1.1870486736297607, 0.3299020826816559, 0.5938827395439148, -0.3616688847541809, 0.4064297676086426, -0.881274402141571, 0.31726253032684326, -1.6430355310440063, -0.06851158291101456, 1.0800492763519287, 0.3661768138408661, -0.5386753082275391, -0.6093199849128723, 1.1045547723770142, 0.9755920171737671, 2.3576765060424805, -0.4837775230407715, 1.246504545211792, 2.022608995437622, -0.5297451019287109, -0.560862123966217, 0.05412410572171211, 0.9447271227836609, 0.4030102789402008, 0.34333357214927673, -1.5933449268341064, 0.08404704928398132, -0.6520775556564331, -0.06339067220687866, -1.9282915592193604, 0.9893285632133484, 0.36571434140205383, 0.07906966656446457, -0.4408225119113922, 0.991121768951416, 1.0876221656799316, 0.7021886706352234, -0.6383882761001587, 0.11275674402713776, 0.3701398968696594, 0.8828790187835693, -0.6583892703056335, 0.3447711765766144, 1.9253301620483398, -1.2689317464828491, 0.6535844802856445, 0.8903228044509888, 0.38533079624176025, 0.2143777757883072, -0.5731050372123718, -0.668518602848053, 0.43514731526374817, -0.5212464332580566, -1.0985103845596313, -0.22215218842029572], [0.6906363368034363, 1.4635878801345825, 0.7326852083206177, 0.7735317349433899, 0.4666048586368561, 1.7017161846160889, 0.906491756439209, -1.4019068479537964, -0.9269092679023743, 0.20358368754386902, 0.5639017820358276, 0.9335504770278931, 0.5430047512054443, -0.5564557313919067, 0.04504978284239769, -1.3141720294952393, 0.6373755931854248, 2.1693062782287598, 0.9257735013961792, -1.104500412940979, 1.3137744665145874, 0.42103588581085205, -1.4755933284759521, 0.5320123434066772, 0.4869793653488159, 2.348322868347168, 2.6096668243408203, 0.40473148226737976, -2.0842106342315674, 0.12729448080062866, 0.6965944766998291, 0.5658770799636841, -1.470504879951477, 0.9425297379493713, -0.10079779475927353, -0.4972164034843445, 0.38620737195014954, -2.239680290222168, -0.9162468910217285, 0.3950723707675934, 1.135541319847107, 0.6521949768066406, 0.02340070530772209, -1.3457432985305786, 0.5112170577049255, 1.047935128211975, -1.5650578737258911, 0.9901357889175415, -0.057958878576755524, 1.2035082578659058, -0.567807137966156, 0.33871719241142273, 0.4721475839614868, -0.5461891889572144, 1.1388404369354248, -1.1266005039215088, -0.06267300248146057, 0.09252134710550308, 1.0631194114685059, -0.40140727162361145, 0.12254177778959274, 0.7005277872085571, -1.9440971612930298, 0.9830511808395386, -0.12758386135101318, 0.4367816746234894, -0.2996915578842163, -0.8027929663658142, 1.1333142518997192, 0.6400079727172852, -1.4381153583526611, 0.6652176380157471, 0.6066632866859436, 0.5807397961616516, 0.03612314164638519, -0.34685760736465454, -1.1460661888122559, 1.484409213066101, 0.752764880657196, 1.6804139614105225, -0.8802592158317566, 1.3924520015716553, 0.35788872838020325, 1.2309074401855469, 0.14889761805534363, 2.1025257110595703, 1.0703297853469849, -0.12379653006792068, 0.02341112308204174, -2.122095823287964, 0.6389555931091309, -0.11934138834476471, 0.1175151988863945, -0.361572265625, 2.1560218334198, -1.0966366529464722, 1.063509225845337, -0.4491914212703705, 1.3123624324798584, 0.6259152889251709, -2.1734085083007812, 1.5709902048110962, -0.0066031343303620815, -0.8897782564163208, -0.1516292542219162, -2.0641674995422363, 0.2685735523700714, -1.726936936378479, 0.0020473951008170843, -0.059816665947437286, 1.072493314743042, 0.8782391548156738, -0.23239921033382416, -1.6020323038101196, 0.36438941955566406, 0.8604729175567627, 1.2069002389907837, 0.22028294205665588, 0.01876685954630375, 1.359450101852417, 1.4230196475982666, -1.0533329248428345, -0.4990091323852539, 0.42845332622528076, -0.528191328048706, 0.5276411771774292, 0.670991837978363, 1.6817303895950317], [0.7814851999282837, 0.9579151272773743, -1.743471384048462, -0.43704953789711, 0.9608619213104248, -0.637272834777832, 1.9658417701721191, 1.0255441665649414, -0.8826216459274292, 0.6319682002067566, -0.4504416882991791, 1.7251238822937012, 1.6775269508361816, -1.5390211343765259, -1.0162569284439087, 1.4403178691864014, -0.6944465637207031, 0.7555939555168152, -0.6173428893089294, -0.6679045557975769, 1.7539271116256714, 0.9665641784667969, 0.997455358505249, -0.16344887018203735, 2.1349713802337646, 1.1031526327133179, -0.43764162063598633, 0.44647690653800964, 0.3190404772758484, 0.5822596549987793, -1.522853136062622, 0.9817051291465759, -1.1793211698532104, -0.8027621507644653, -1.8279386758804321, 0.49697059392929077, 0.7198778986930847, 2.9195823669433594, -1.2283297777175903, 1.1129014492034912, 0.13372644782066345, 0.3796927332878113, 0.7296591401100159, 0.3458618223667145, 0.5462231636047363, -0.06324509531259537, -1.2534970045089722, 0.40542709827423096, 0.8428739905357361, 0.5600132346153259, 0.7788794636726379, -0.05090494081377983, -1.4910550117492676, -0.4657559394836426, -0.6269223690032959, 1.2319520711898804, -0.13261863589286804, 0.007807028945535421, 0.2441738247871399, 1.7373111248016357, 0.4981499910354614, 0.049355797469615936, 0.24630531668663025, 0.2612394392490387, -1.459193229675293, 1.0337578058242798, 1.3265565633773804, -0.12539198994636536, 1.638235330581665, -0.02192213386297226, 0.02105989120900631, 0.013723615556955338, 1.2303987741470337, 1.626407504081726, 0.3215619623661041, 0.7996739745140076, -0.47732922434806824, 0.18122407793998718, 1.8127315044403076, 0.6681591272354126, -0.44620105624198914, 0.5967647433280945, -0.17498897016048431, -0.3190736174583435, 0.1955123096704483, 0.1667749285697937, 0.9808434844017029, 0.40594708919525146, 0.06790940463542938, -0.32606953382492065, -0.543584406375885, -0.5475950837135315, 1.427082896232605, -1.6800837516784668, -0.03129152953624725, -0.9236361384391785, -3.4787614345550537, -1.8122004270553589, 0.5853309631347656, -0.2200053334236145, -0.24190907180309296, -1.1090137958526611, -0.8353268504142761, 0.42199352383613586, -0.4712119400501251, 0.6556161642074585, 2.7962357997894287, -0.663687527179718, 1.0835589170455933, -0.016624508425593376, 0.8873729109764099, 0.7692455649375916, -0.6674069762229919, -0.540645956993103, 0.17815490067005157, 0.016453152522444725, 0.06019490584731102, -1.7603228092193604, 0.5545745491981506, 1.6985429525375366, -0.4414074420928955, 0.3439847230911255, 1.2124437093734741, 0.9245179295539856, 0.5327108502388, -2.490004301071167, 0.4642161726951599, -1.6767646074295044], [-0.9837774634361267, 0.5387794375419617, 0.7946959137916565, -0.08437465131282806, -1.1679890155792236, 1.230857253074646, 1.5344009399414062, -1.729162335395813, 0.4413378834724426, 1.5745924711227417, 1.3444105386734009, -0.19646891951560974, 0.812088668346405, -0.817263126373291, -0.7460572719573975, 1.725645661354065, -1.1502830982208252, 0.600782036781311, 0.3649385869503021, -0.49836465716362, 0.1654425412416458, 1.5379053354263306, 0.3669579327106476, -0.8026655316352844, -0.5844297409057617, -1.7564353942871094, -0.6044775247573853, -0.6832722425460815, 0.882870078086853, -0.6708675026893616, -1.1247481107711792, -0.2536989450454712, 0.17755702137947083, 0.7564527988433838, 0.9783036708831787, 0.8443271517753601, 1.2194793224334717, -0.8411509394645691, 0.6890304088592529, -0.07106412947177887, 0.9755902290344238, 0.6995198130607605, 0.4628952145576477, -0.09234698116779327, 0.09991611540317535, 0.8851719498634338, 0.5923153758049011, 0.5292711853981018, -0.013728086836636066, -0.32631805539131165, -0.8737662434577942, 0.2606232762336731, -1.1919567584991455, 1.553001880645752, 0.09215854108333588, 0.6416468024253845, -0.48835793137550354, 1.205021619796753, -0.43941566348075867, 1.5175944566726685, -0.2699657380580902, 0.05297054350376129, 0.6618666648864746, -0.19354936480522156, 1.386527419090271, -0.7708734273910522, 1.0435467958450317, 0.8734093308448792, -0.13252165913581848, -0.6086121201515198, -0.6741824746131897, -0.22287042438983917, 0.17706415057182312, -0.2540648877620697, -0.42124301195144653, -0.24515904486179352, -1.2849609851837158, -1.0164247751235962, 0.5892243981361389, 1.3050997257232666, -0.3875740170478821, 0.6486984491348267, -0.737540066242218, 1.2997138500213623, -1.3520054817199707, 1.8719688653945923, -0.7933713793754578, 1.2340822219848633, 0.472206711769104, -0.8291341662406921, -0.3652055859565735, 0.7871817946434021, -1.2533555030822754, -0.17270062863826752, -0.941015899181366, -0.27108192443847656, -0.052363600581884384, 0.8444260358810425, 0.3232574760913849, 0.9911225438117981, 1.2031317949295044, -0.5627318024635315, 0.6060361862182617, -0.021045144647359848, -0.7553588151931763, 0.972697377204895, 0.8727817535400391, -0.31352749466896057, -1.2904613018035889, 0.8199365139007568, 0.2721622884273529, -0.34203776717185974, 1.23099684715271, 1.42641019821167, -0.5114330053329468, -0.46372511982917786, 2.580263614654541, 0.7545417547225952, 1.6130996942520142, -0.22289182245731354, -1.5350912809371948, 0.8864641785621643, 0.04690433293581009, -0.5441312193870544, 0.9218274354934692, 0.501300036907196, -0.8054908514022827, 1.1401705741882324]]], [[[0.7095609903335571, -0.3100678026676178, 0.02548939548432827, -0.344625860452652, -0.14295068383216858, 1.397107481956482, 2.2064664363861084, -0.19431452453136444, 1.6592422723770142, -2.0094351768493652, 0.0024937093257904053, -1.0304322242736816, -0.6677717566490173, -0.35297101736068726, 0.5359212756156921, -1.8288315534591675, 1.6187727451324463, 0.4192913770675659, 0.09636656939983368, -0.9016150236129761, 0.11111829429864883, -1.622718334197998, 0.8482100963592529, -1.6371970176696777, 0.4861307442188263, 0.5023747086524963, -0.34841224551200867, -0.8790308237075806, 0.44404518604278564, -0.9981102347373962, 0.2348157912492752, 0.6981332898139954, -0.14943836629390717, 2.245142698287964, 1.435221791267395, -1.5515785217285156, 0.2518267035484314, 1.5382394790649414, 0.06649217754602432, -1.2013237476348877, 0.3398802578449249, -0.3340010344982147, 0.5053480863571167, 0.2497130036354065, -0.44873300194740295, -0.4223436713218689, 0.03407580032944679, 0.5573023557662964, -0.9365904331207275, -0.42767345905303955, 1.349822998046875, -2.1492419242858887, -0.45919761061668396, -1.2981878519058228, -0.712800145149231, -1.694770336151123, 0.21853893995285034, -0.5448921322822571, 1.4800657033920288, 0.9462641477584839, 0.9267970323562622, 0.4367235004901886, 1.2326197624206543, -0.6682659983634949, 0.0018998711602762341, -1.8276275396347046, 1.2109116315841675, 0.06370221078395844, -1.0291649103164673, 0.38700398802757263, 0.4842725694179535, -1.4250248670578003, 0.48081934452056885, -1.3450285196304321, 1.248300313949585, -1.5581589937210083, 0.4986182749271393, 0.3669968545436859, 1.6318858861923218, 0.8474560976028442, -0.5370607972145081, 0.09815066307783127, -1.6179354190826416, 1.9589084386825562, -0.07900083065032959, -0.06494121253490448, -1.0047377347946167, -0.3266131281852722, -1.119523286819458, 0.08148818463087082, 1.1725186109542847, -0.10605094581842422, 1.3018473386764526, 1.1407610177993774, -1.1686203479766846, -0.7290371656417847, 0.781297504901886, 2.4660701751708984, 1.0747289657592773, 1.9977599382400513, 0.7509638667106628, 1.7418774366378784, 1.8748128414154053, 0.6137615442276001, -1.4679168462753296, 0.9247952699661255, 1.0967832803726196, 0.49485427141189575, 0.9803107976913452, -0.7222751975059509, -0.6398507952690125, 1.5904518365859985, 1.4342155456542969, 1.4536724090576172, -0.3162442743778229, 0.41459986567497253, 0.1352739930152893, 0.9057160019874573, 0.3303523361682892, 1.6239436864852905, -0.7629119157791138, 0.5608347654342651, -0.7273001074790955, -0.594033420085907, -0.04574596509337425, 0.2700200378894806, 1.7508149147033691, -0.5676645636558533], [-0.21157503128051758, 0.7246447801589966, -0.03525139018893242, -0.7885043621063232, 0.3729645907878876, -0.14784783124923706, 1.837827444076538, 1.4747122526168823, 0.4959907829761505, -0.4660688638687134, -1.855728030204773, 0.5144109725952148, -0.14419357478618622, -0.04151706397533417, -0.2597874402999878, -0.5183874368667603, -0.9656013250350952, 1.0386286973953247, 0.4242595136165619, -1.3642185926437378, 0.10514860600233078, 0.9913772940635681, -0.7554590106010437, -0.3643505573272705, 0.8229019641876221, -0.050379056483507156, 0.6879026889801025, -0.8663199543952942, -0.0876118615269661, -1.9639842510223389, -2.0757360458374023, -0.427918404340744, 0.5530077815055847, -1.7410132884979248, -1.23101007938385, 1.555031418800354, 1.3203879594802856, 1.2101048231124878, -0.4626675546169281, -0.6466664671897888, 0.9608833193778992, -0.1272413283586502, 0.4526243209838867, -0.5283567309379578, -1.3577475547790527, -2.1231791973114014, -0.23298878967761993, 0.32672345638275146, -1.4596855640411377, 0.10143397003412247, -0.8705399036407471, 0.11117946356534958, -0.0684567466378212, -1.1210459470748901, -0.0033523780293762684, -0.3428342938423157, -0.45549964904785156, -0.8268100619316101, -0.6020504832267761, 0.6745844483375549, 0.6602571606636047, -1.1392806768417358, -0.0011795469326898456, 0.1198834776878357, -0.813653826713562, 0.40010806918144226, 0.5576699376106262, 1.8578742742538452, -1.575183629989624, 0.53602534532547, -0.7010542750358582, 0.3627058267593384, 2.1753244400024414, 0.39457055926322937, -1.2124600410461426, -0.541968047618866, 0.6899423599243164, -1.6103205680847168, -0.2075660526752472, -2.2712385654449463, 1.098285436630249, 0.37535983324050903, 1.5884414911270142, -0.6400452256202698, -0.8882083296775818, 0.6009417772293091, 0.5782752633094788, -1.2164400815963745, 0.17207182943820953, -0.9471033215522766, -0.9832548499107361, 0.8024178147315979, 0.733158528804779, -1.0191618204116821, 0.3007884621620178, 0.7971322536468506, -1.0189387798309326, 1.3021186590194702, -0.6081159114837646, -1.7710273265838623, -0.8409290313720703, -0.1810731440782547, -1.6557427644729614, 1.7963018417358398, 0.4095107316970825, -1.5046272277832031, 0.9581727385520935, 1.087716817855835, -0.10969409346580505, 0.6800172328948975, 0.9820102453231812, 1.0117990970611572, -1.7114540338516235, -0.08422813564538956, -1.2136716842651367, 0.020055681467056274, -1.2837412357330322, -1.2897857427597046, 0.7639549374580383, -0.20584669709205627, -1.0981426239013672, 1.3961687088012695, -0.6708041429519653, 2.2066831588745117, -1.18607759475708, -0.9073439836502075, -1.040767788887024, 0.7883951663970947], [-2.560645818710327, -1.8768095970153809, -0.574080228805542, 0.8041850328445435, 0.7010823488235474, -0.7649420499801636, -1.0302373170852661, -0.06440266221761703, 1.0349727869033813, 0.048932548612356186, 0.5826547145843506, -1.427951693534851, -0.48388272523880005, 0.9298500418663025, 0.46860218048095703, 0.22450268268585205, 0.054961416870355606, 0.8184080123901367, -0.45240432024002075, -0.6253446340560913, 0.06539148092269897, -0.90373295545578, -1.7590752840042114, -0.3550858497619629, -1.3064236640930176, -0.006910952273756266, 1.0065730810165405, -0.7990081906318665, -0.15061314404010773, -0.36097443103790283, 0.4712299406528473, -1.2728193998336792, 0.4445502460002899, -1.6296442747116089, 0.027220968157052994, 0.7680389285087585, -2.3335745334625244, 1.9680438041687012, -1.92465341091156, 0.9386591911315918, 1.8637721538543701, -1.038041591644287, -0.7822692394256592, 0.9023188352584839, -0.2968488335609436, -2.3772389888763428, 1.2041311264038086, 1.2213611602783203, 0.5823461413383484, -0.593573808670044, -1.1877903938293457, 0.4425652027130127, 0.9154548645019531, -0.8670209050178528, 0.5874084234237671, 0.05793910101056099, 0.7779324054718018, -1.1463041305541992, -0.5186182856559753, 0.44887253642082214, -0.15160097181797028, 0.25976982712745667, -1.8611069917678833, -1.2678062915802002, -0.6916516423225403, 0.3867485821247101, 0.15536761283874512, 1.368916392326355, 1.9059414863586426, -0.35405421257019043, 0.3737228810787201, 2.1395132541656494, 0.353371262550354, 0.9014610052108765, 0.8159493803977966, 0.020991511642932892, 1.2933018207550049, -1.8732118606567383, 1.5871700048446655, -1.7188172340393066, -0.13680051267147064, 0.230804905295372, -1.0159834623336792, -2.0100324153900146, -0.6781928539276123, -0.5371352434158325, 0.7657476663589478, -0.4856128692626953, 0.7585217356681824, -0.4163074493408203, -1.2853963375091553, -0.04106632620096207, -0.720342218875885, -1.2765923738479614, 0.7949714064598083, -0.9223031401634216, 0.15320327877998352, -1.3164052963256836, 0.918271005153656, -0.7048671245574951, -0.5689424872398376, -1.0484678745269775, -1.2529999017715454, -1.3850051164627075, 0.5066596865653992, -2.0092947483062744, 1.0117496252059937, -2.8092851638793945, -2.439206600189209, -1.6511142253875732, -0.5652646422386169, -0.39075639843940735, -1.6014362573623657, 1.3407281637191772, -2.1955924034118652, -0.7229527235031128, 0.7329875230789185, -0.7545042037963867, 0.06975012272596359, -1.382433533668518, -0.06659014523029327, 0.1116010844707489, 0.9964025616645813, 0.32936620712280273, 0.26373106241226196, 0.377518892288208, -2.4138829708099365, 0.144968643784523], [-0.21996240317821503, 0.4839319586753845, -0.8191623091697693, 1.2420226335525513, 0.6238437294960022, 0.8288214206695557, 1.6779192686080933, 1.2034848928451538, -0.13617496192455292, -1.0134872198104858, 1.2698549032211304, 2.621023654937744, -0.9106560349464417, -0.9471918344497681, 0.13883577287197113, 0.21991485357284546, -0.18967905640602112, 0.6241891980171204, 0.3304939866065979, 0.9150122404098511, 0.6411035656929016, -2.5392000675201416, -0.9650190472602844, -2.188643455505371, 1.576101541519165, -0.8217648267745972, -0.3141951858997345, -2.4085376262664795, 0.36830389499664307, -0.7347213625907898, 1.2607735395431519, -0.980608344078064, -1.3320729732513428, 0.8660256266593933, -0.6353670954704285, -0.35042300820350647, -1.126863956451416, -0.8561150431632996, 0.031273484230041504, -0.5716098546981812, 1.16032874584198, -0.816666841506958, 0.34876924753189087, 0.32063573598861694, 0.19972053170204163, 1.0023976564407349, -0.6187996864318848, -0.03878433629870415, 0.20213907957077026, -1.3839746713638306, 0.5534287095069885, 1.1128016710281372, -0.6356409788131714, -1.0966320037841797, 0.04468469321727753, 0.008940390311181545, -0.9729533195495605, -0.6543021202087402, 0.49991676211357117, -0.04948442056775093, -0.9233524799346924, 1.386580228805542, 1.168082356452942, -0.29977768659591675, 1.038026213645935, -0.9035940766334534, -0.4080961346626282, -1.156294822692871, 0.40972164273262024, 0.1985684037208557, 0.484274297952652, -0.7322864532470703, 1.2994893789291382, -1.3924143314361572, -0.7223258018493652, 0.26162198185920715, 0.526932954788208, -0.9580810070037842, 0.6020072102546692, 0.27208444476127625, -1.1973515748977661, -1.025638461112976, -0.6146632432937622, 0.06477976590394974, 0.6115216016769409, -1.0904009342193604, 0.1029900535941124, 0.8447807431221008, -0.09267120808362961, 0.42731404304504395, 0.010670226998627186, 0.040202800184488297, -0.05231394246220589, -0.5626668334007263, -0.3708212971687317, 1.698833703994751, -0.9114472270011902, -0.8022010922431946, -0.01848961040377617, -1.2367236614227295, -0.11161531507968903, 0.45701146125793457, 0.5624948740005493, 0.652077317237854, -0.9962493181228638, 0.14808660745620728, -0.5354242324829102, 1.9950579404830933, 1.8804020881652832, -0.6109813451766968, 0.4887455403804779, -0.25936830043792725, -1.254624605178833, -0.029321104288101196, -0.22034962475299835, 1.3429388999938965, -0.9319255352020264, -0.6541143655776978, 0.3969924747943878, 0.547258198261261, -1.441084623336792, -0.8608658313751221, 1.8103290796279907, 1.3354572057724, -0.4384866952896118, -1.7143033742904663, 0.7639206051826477, -0.1946375072002411], [0.33515071868896484, 1.4950443506240845, -0.620689332485199, -0.9006885290145874, 1.6631501913070679, -0.937069833278656, -1.392927646636963, 0.00784364901483059, 0.8619669079780579, -0.8753177523612976, 0.1462676227092743, 0.28703415393829346, -0.6794686317443848, -0.6103346347808838, 0.8025175929069519, -2.14689040184021, 1.2768617868423462, -0.7226853370666504, -0.7860518097877502, 0.5923957228660583, -0.7835215926170349, 0.772018551826477, 0.43433836102485657, -1.4339348077774048, -1.1396194696426392, -0.5010199546813965, 2.6410067081451416, -1.6799012422561646, -0.0474918968975544, -0.7207555174827576, 0.5463778376579285, 0.874193549156189, 0.7408170700073242, 0.08447924256324768, -0.8937423229217529, -1.6956130266189575, -0.9756563305854797, 0.1505577564239502, 1.1484531164169312, 0.9963087439537048, 0.2769124507904053, 0.10031091421842575, 0.21837402880191803, -0.29241666197776794, -0.3438027501106262, 1.0062364339828491, -0.986008882522583, -0.8332789540290833, 0.2920582890510559, 2.441265106201172, 0.39518657326698303, -0.022613724693655968, 1.811614751815796, -0.17164306342601776, 0.1933802217245102, -1.5360536575317383, -0.5098064541816711, -1.4382604360580444, 1.945071816444397, -1.5070730447769165, 1.0124856233596802, 1.2648416757583618, 1.3388937711715698, -0.14768506586551666, -0.05786745995283127, -1.8237935304641724, 0.6366629600524902, 2.0504000186920166, 0.5829384922981262, -0.8122042417526245, 1.0202052593231201, 0.25026270747184753, -1.4208660125732422, 0.3543013632297516, 0.44479572772979736, -1.101549744606018, -0.0037625099066644907, -0.09027904272079468, -0.7501217722892761, 0.7421330213546753, 0.0956350713968277, 0.7576484084129333, -1.191959023475647, 0.6565957069396973, -0.5460116267204285, -1.485467553138733, -0.8257168531417847, 1.0372403860092163, -0.16714884340763092, 0.14868998527526855, 0.5354427695274353, 0.007928377017378807, 0.5136297941207886, -0.4652785658836365, 0.12942476570606232, -0.8225241303443909, -0.47721585631370544, 1.0772364139556885, -1.0602986812591553, 0.08271081745624542, -1.1735447645187378, 0.2985077500343323, -0.7898061871528625, -1.2103629112243652, -0.037252604961395264, -0.8441043496131897, 0.47526124119758606, 0.5895938873291016, -0.5516898036003113, 0.4709448516368866, -0.544095516204834, -0.11308562010526657, 0.30079150199890137, -1.0729014873504639, -0.4124387800693512, -0.30763742327690125, 0.4927518367767334, 0.19041231274604797, 0.10283038765192032, 0.4358896315097809, 1.1554690599441528, 1.3281357288360596, -0.5268080830574036, -1.5642919540405273, -2.312021493911743, -0.08481103181838989, 1.4098994731903076, -0.18351712822914124], [-0.11819427460432053, 0.31429561972618103, 0.30525410175323486, -0.8893412351608276, -0.619070291519165, 1.6407321691513062, 1.6311246156692505, 1.051984190940857, -1.0741655826568604, -1.0490964651107788, -0.5747627019882202, 1.259600281715393, 1.303827166557312, -0.4004456102848053, 0.09098855406045914, 0.6235876679420471, 0.4181317389011383, -1.0472462177276611, 0.3646472692489624, -1.2847157716751099, -0.21674282848834991, -0.46956807374954224, 1.149760127067566, -0.7609553337097168, 0.49898287653923035, 0.5417127013206482, -0.07964634895324707, -0.42111876606941223, -0.4609795808792114, 0.9502678513526917, -1.3414623737335205, 1.5875097513198853, 0.39837926626205444, 0.8312460780143738, 0.17849642038345337, -0.4652954638004303, -0.8002205491065979, 0.7941730618476868, -1.5436550378799438, 0.34768182039260864, -0.6622060537338257, -0.8051729202270508, 0.09601855278015137, -0.5313941836357117, 1.3796882629394531, 0.6739329099655151, -0.10585134476423264, -1.5543887615203857, 0.06280197948217392, 0.9058328866958618, -1.0079249143600464, 0.8042190670967102, 0.0321565717458725, -0.2274010181427002, 0.4525705873966217, -0.6846917271614075, 0.7652007341384888, -1.1797747611999512, -0.3315947651863098, -1.4867446422576904, -1.051770806312561, -0.9795367121696472, 0.3400909900665283, -0.7492274641990662, -0.6305947303771973, 0.7404892444610596, 0.9832180738449097, 0.12095239758491516, -0.8468499183654785, -1.6366896629333496, -1.0900311470031738, -0.848784327507019, -0.5456636548042297, 0.6894954442977905, -0.1476617157459259, 0.039985980838537216, -1.08001708984375, -1.0096495151519775, -0.04000215232372284, 0.2892446517944336, -0.37410154938697815, -1.0697048902511597, -1.8925316333770752, 0.7126491665840149, -1.2702146768569946, 2.257922887802124, 1.90378999710083, -1.768796682357788, 1.2782137393951416, 0.6562036871910095, -0.6215267777442932, 0.31453341245651245, 1.3671319484710693, 2.111868381500244, 1.66818368434906, -0.36840009689331055, -0.6458487510681152, -0.783065676689148, 0.003344569820910692, 0.12050226330757141, -0.02981267124414444, 0.3096218407154083, -3.2172234058380127, -1.8330137729644775, -1.6327251195907593, -0.49126940965652466, -1.118838906288147, -1.3699008226394653, 0.557948112487793, 0.12148021906614304, -1.6864217519760132, 0.8606079816818237, -1.0668981075286865, -0.2641725540161133, 1.128883957862854, 0.4404810667037964, 1.4061195850372314, -0.2176552712917328, 0.7146803140640259, 0.505712628364563, -1.86049222946167, 0.17061366140842438, 0.2975027859210968, 0.0329466238617897, -0.6349102258682251, -1.66975998878479, 2.207792043685913, 0.3876461386680603], [0.1440170556306839, -0.07514727860689163, 0.4485699534416199, -0.5766756534576416, -1.0785342454910278, -1.243464708328247, -0.21760155260562897, -0.3123750686645508, -0.35253363847732544, 0.22889076173305511, 0.21391800045967102, 1.0951722860336304, -1.4607490301132202, 0.6049829125404358, 1.292194128036499, 0.0024981764145195484, 0.45138660073280334, -0.04118593782186508, -0.6646583676338196, 0.38711580634117126, -0.13683784008026123, -2.570152521133423, 0.8062478303909302, -0.8914728760719299, -0.7805073261260986, -2.066555976867676, 0.05063126981258392, 0.828193724155426, 0.611775279045105, 1.5871758460998535, -0.20620937645435333, -0.040936242789030075, -0.5196242332458496, -0.6410073637962341, 1.1195380687713623, 1.6198952198028564, -0.2647032141685486, -0.0838615894317627, -1.1994415521621704, -0.5678105354309082, -0.6986167430877686, -1.511724591255188, 0.47777947783470154, 0.740787148475647, -2.3830411434173584, 0.0031430174130946398, -0.02993060275912285, -1.9359846115112305, 1.362626075744629, 1.4486619234085083, 0.22517108917236328, -0.10458873957395554, 0.13389718532562256, 1.2692201137542725, -0.7298815846443176, -0.16301250457763672, 1.2166980504989624, -1.4981074333190918, 0.05561006814241409, 0.8925838470458984, 0.7683516144752502, 1.3518842458724976, 0.1082841157913208, 0.5296574831008911, -0.3083190321922302, -1.1912871599197388, -0.26482558250427246, 0.7994507551193237, -0.03967323526740074, 0.3235136866569519, -1.7408809661865234, 0.6551970839500427, -3.0315065383911133, 0.1537226140499115, 0.3583691120147705, 2.6992204189300537, -0.20573323965072632, 0.7351101636886597, -0.7727293372154236, 0.3892386555671692, 0.007016806397587061, -0.4639246165752411, 1.5124619007110596, 0.2210320234298706, 0.3248857259750366, 0.4540165662765503, -1.0795305967330933, 0.5092954039573669, 0.205744668841362, 0.40337201952934265, -0.28319790959358215, -0.021228203549981117, 0.5007227063179016, 0.45898306369781494, -0.09382389485836029, 1.1092866659164429, 1.5851715803146362, -0.4422692060470581, 1.0704095363616943, -0.38607755303382874, -0.3402906358242035, -1.4268507957458496, 1.231899380683899, 0.3422594964504242, -0.3048693835735321, 0.9608732461929321, -0.12338338792324066, -0.010138672776520252, 0.04297441616654396, -0.0731513574719429, 0.47115975618362427, -1.2174689769744873, 0.3404582142829895, -0.9474352598190308, 0.35861146450042725, 1.7221266031265259, -0.46112096309661865, 0.7742409706115723, -1.4173253774642944, 0.9403384327888489, 1.3593477010726929, -1.0028495788574219, 1.4572001695632935, -0.09826122969388962, -0.5131340622901917, 0.6150616407394409, -0.6769888401031494, -1.6195200681686401], [0.673835039138794, -0.7360211610794067, 1.2776305675506592, -1.1303839683532715, -1.00129234790802, 0.025185130536556244, -0.903891921043396, -1.2450717687606812, 0.6875767707824707, 0.4159949719905853, 1.1921311616897583, -0.5175303220748901, -0.10968701541423798, 0.8728147745132446, 2.198654890060425, 0.6493366360664368, 0.3601679801940918, -0.05922409147024155, 0.03685491904616356, -1.666140079498291, -0.5448912978172302, -0.1201757863163948, -0.24287284910678864, 0.4981124699115753, -0.5717028975486755, -1.4400802850723267, -0.26956290006637573, 1.9840079545974731, -0.5630086660385132, -0.5446567535400391, 0.5359646081924438, 0.725813627243042, -0.5655882954597473, 1.0629581212997437, -0.11990097910165787, -0.6614517569541931, -0.462155282497406, -0.039869584143161774, 0.5401323437690735, -0.3468705415725708, 0.48843199014663696, -0.9218423962593079, 0.04844330623745918, 0.05532817915081978, 1.7883636951446533, -0.007032702211290598, -0.3107101321220398, 0.32345977425575256, 1.192723035812378, -0.3772053122520447, -0.07611696422100067, 1.393358826637268, -1.0326746702194214, 1.0978937149047852, -1.0612696409225464, -1.7222378253936768, -1.0915669202804565, -0.45970386266708374, 0.09798367321491241, -0.6566680669784546, 0.5257764458656311, -1.3465466499328613, 2.372725009918213, -1.5263527631759644, 1.0222539901733398, -1.636419653892517, 0.36644625663757324, -1.1810917854309082, -0.2514619529247284, 0.8007935881614685, -0.05345752835273743, 0.682131290435791, 0.009865084663033485, 0.5550448298454285, -0.8461341857910156, 0.3841506838798523, -0.24108034372329712, 1.3125152587890625, 1.1283763647079468, 0.06908675283193588, -0.9728766679763794, -0.8978536128997803, -1.531329870223999, -0.6183552145957947, 0.040089014917612076, -0.7017072439193726, -0.33828091621398926, 1.366507887840271, 0.023706207051873207, -0.4441157579421997, -1.3520818948745728, -1.5257245302200317, -2.0308115482330322, -2.040841579437256, 0.1637442260980606, 0.19102077186107635, 0.5625642538070679, 1.6221369504928589, 0.21194083988666534, 1.227211833000183, -0.037879861891269684, -1.3486602306365967, 0.9541476964950562, -1.3493205308914185, -0.348054438829422, 1.252272129058838, 1.632948637008667, -0.6821085810661316, -0.33921104669570923, 1.323616623878479, -0.21707262098789215, -0.747327446937561, 0.5991040468215942, -1.6530576944351196, -0.961591899394989, 1.2568162679672241, 0.14743837714195251, 0.5441649556159973, -1.7533624172210693, 0.9610776305198669, -0.725421130657196, 0.6035835146903992, 1.9459946155548096, 1.3646959066390991, -1.3433220386505127, 0.47024115920066833, -1.6912320852279663, 0.02954866923391819], [0.5377479791641235, -0.4523489475250244, 0.06736057996749878, -0.461322158575058, -0.1489885151386261, -1.4641886949539185, -1.0388182401657104, 0.7775005102157593, -0.6850377917289734, -1.5142117738723755, 0.008662927895784378, 0.6827517151832581, 0.04431305453181267, -1.4281715154647827, 1.9956859350204468, 0.4064348042011261, 2.2120959758758545, -0.3932693898677826, -0.6735345721244812, 0.13513562083244324, 0.10814517736434937, -1.0715103149414062, 0.13817113637924194, -0.01289253868162632, -1.3309131860733032, 0.533886730670929, -0.9357141852378845, -0.782511830329895, 0.3221982419490814, -0.4922088086605072, -0.29554489254951477, -0.2461017668247223, 0.18155746161937714, -0.4666477143764496, -0.872210681438446, -0.2224833220243454, -0.1206907406449318, 1.1262903213500977, 0.7700014114379883, -0.4610595703125, -1.048429012298584, -0.2821306884288788, -0.6112462878227234, -0.5584007501602173, 0.30711430311203003, 2.093402147293091, 0.02968742698431015, -1.2786763906478882, 1.2448869943618774, -0.2951907813549042, -1.4935907125473022, -1.2610719203948975, 0.5835263133049011, 0.4692610800266266, 0.11542298644781113, 0.1440928727388382, -2.3353004455566406, -1.10813307762146, 0.31498560309410095, -0.6896743178367615, -1.0868942737579346, -0.5582907199859619, 1.7736384868621826, -1.7575868368148804, 0.5566301345825195, 0.5695945024490356, -0.5629624128341675, 0.8397612571716309, -1.1814496517181396, -0.2305588722229004, 0.9737185835838318, -2.0289604663848877, -0.7539339661598206, -0.05429365485906601, 0.5916967988014221, 0.2770633399486542, -0.7066303491592407, 0.27193859219551086, 1.7124158143997192, -0.5071701407432556, -0.8589391112327576, 2.4997050762176514, -0.3068436086177826, -0.17494583129882812, 0.3157326579093933, 1.5531271696090698, 0.45110467076301575, -0.2595561742782593, -1.3660274744033813, 0.1580977439880371, -0.28293612599372864, 0.20173805952072144, 0.1801462322473526, 0.15181668102741241, 0.02070450782775879, -0.9117680191993713, 1.4605224132537842, 1.501613736152649, -0.04989419877529144, 0.023000478744506836, 0.3424730896949768, -0.5557767748832703, 0.09764847904443741, 0.79109787940979, -0.2408832609653473, -0.012057581916451454, -1.0740858316421509, -0.8678178787231445, -2.3986597061157227, -1.8263541460037231, -1.9468470811843872, -0.018481165170669556, 0.8962650895118713, 1.8638285398483276, -0.5847963690757751, 0.014810120686888695, -0.5544499158859253, -1.2552073001861572, 0.7384395599365234, -1.357608675956726, 0.011032507754862309, 1.4410221576690674, 0.6297910809516907, 0.9981434345245361, 0.11631569266319275, 0.9928745627403259, 0.2578444480895996, 0.3511494994163513], [-0.7943165302276611, -0.8535111546516418, 1.2431881427764893, 0.7629818916320801, 0.4414006173610687, -2.321087121963501, 0.9736266136169434, 1.3834751844406128, 0.631807267665863, 0.5336431860923767, -1.0914347171783447, -0.5592145919799805, -1.6989022493362427, 0.04206427186727524, 0.7192460298538208, 1.3349359035491943, -2.3064517974853516, 0.17763233184814453, -1.0517529249191284, -0.5528113842010498, -1.2306472063064575, 0.29713961482048035, 0.7351592779159546, -0.822704553604126, -0.5186630487442017, 1.0537060499191284, -0.5849130153656006, -0.4361293315887451, -1.7074134349822998, 0.0011472493642941117, -0.5294129252433777, 0.11738108843564987, 0.7877498269081116, -0.7257717847824097, 0.14229343831539154, -0.6154451370239258, 0.6798392534255981, -0.5962422490119934, 1.536906123161316, 1.216591477394104, 1.1443341970443726, 0.5754926204681396, 1.4850856065750122, 0.3978905975818634, -0.18917493522167206, 1.2984156608581543, 1.7328441143035889, 0.16008156538009644, 0.23412233591079712, -0.22822372615337372, 1.6467814445495605, -1.2013990879058838, 1.1653242111206055, -0.8526560664176941, -0.30452761054039, -1.093827724456787, 1.258953332901001, 0.30909043550491333, -1.5007375478744507, -0.31701380014419556, 0.608463704586029, 1.4475725889205933, 0.35045287013053894, 0.04532020911574364, 0.36995160579681396, -1.1489022970199585, -1.4738047122955322, 0.9169731736183167, -0.3715304732322693, 0.022362850606441498, -0.14883999526500702, 0.3059680461883545, -1.7386314868927002, 0.008989084511995316, 0.3474162220954895, 1.2928721904754639, 0.5185934901237488, 1.725831389427185, -0.5434185862541199, 0.6367554664611816, 0.6126599311828613, 0.7959452867507935, 0.7926507592201233, 0.4713563323020935, 0.052872639149427414, 0.7605148553848267, 0.3046441078186035, 0.13602425158023834, 0.600701093673706, -0.48630291223526, 1.7773926258087158, -0.07070871442556381, 0.1562303751707077, 0.13535986840724945, 0.3867402970790863, -0.3905799090862274, -0.40194225311279297, 2.0978832244873047, -1.0305684804916382, 0.4177711009979248, 0.2730181813240051, -1.4896105527877808, 0.6586450934410095, 1.8983873128890991, 0.27861905097961426, 1.0851885080337524, -0.008533860556781292, -0.3563727140426636, -0.2900755703449249, 0.6031640768051147, -1.1771965026855469, -0.8666424751281738, 0.1130501851439476, 0.7426039576530457, 1.7084808349609375, -1.934600591659546, -0.9408479332923889, -0.5645405650138855, -2.1673336029052734, -0.06250544637441635, -1.2071014642715454, -1.402404546737671, 0.772983729839325, 1.2712465524673462, -0.8635595440864563, -1.0567089319229126, 0.8156932592391968, 1.5315297842025757], [0.5891594886779785, 0.6298246383666992, -0.8520123958587646, -1.270084261894226, 1.6527080535888672, -0.15277498960494995, 2.173281669616699, 0.3114703297615051, 0.11180286109447479, -0.39002105593681335, 0.22668111324310303, 0.17720723152160645, 1.8909517526626587, -0.7173363566398621, -0.4824674129486084, -2.164160966873169, -0.637537956237793, -2.367856740951538, -0.2952437996864319, -0.014101123437285423, -0.9499626755714417, -0.8368223905563354, 0.7042086720466614, 0.5344977378845215, 0.14401641488075256, 0.2964337468147278, 0.9096669554710388, 1.2350906133651733, 1.2394570112228394, -0.13862557709217072, -0.01786944270133972, 0.9560287594795227, 0.2507995665073395, -2.244602918624878, -1.0154860019683838, -1.937677264213562, 0.03473745286464691, -0.4010012745857239, 0.18320882320404053, -1.9459527730941772, 0.8347151875495911, 0.6128685474395752, -0.6055372357368469, -0.2710530161857605, 0.13442480564117432, 0.4819851517677307, 0.5888282060623169, -0.5785173177719116, -0.7390173077583313, -0.7722538113594055, 0.12136021256446838, 1.2532436847686768, -0.8072003722190857, 1.6806222200393677, -0.5769597291946411, -0.28495779633522034, -0.3614748418331146, 0.9237882494926453, 0.7479385137557983, 1.1851401329040527, 0.36703234910964966, -1.518342137336731, -0.9739375710487366, -0.009151875041425228, 0.3985797166824341, -0.7707499265670776, 0.26564136147499084, 2.690263271331787, -1.4004019498825073, 0.693216860294342, 1.3135720491409302, 0.00397476926445961, 0.5763620734214783, 0.4881746172904968, -0.7511872053146362, 2.261753797531128, 0.17979660630226135, 1.5796153545379639, -0.32423314452171326, 0.5106306672096252, -1.8447023630142212, -0.1899217814207077, 0.10340673476457596, -1.2521135807037354, -0.6591009497642517, 0.10149599611759186, 0.40883341431617737, -0.9380991458892822, -1.2890594005584717, -1.656933069229126, 0.7119375467300415, 1.3348368406295776, -0.31051239371299744, -0.8185584545135498, -0.5347715616226196, -0.30293112993240356, 0.8956692218780518, 0.16649459302425385, 0.23785360157489777, 0.009635079652071, 1.0527536869049072, 0.9581314921379089, -0.35909074544906616, 1.379838466644287, -0.37113794684410095, 0.9883220195770264, 2.267883062362671, -1.526382327079773, -0.21106280386447906, 0.6645028591156006, -0.26820313930511475, 0.7585324645042419, 0.07774434238672256, 0.619756817817688, 1.5264922380447388, -0.850527822971344, 0.3416471779346466, -0.6410987377166748, -0.3481947183609009, -0.846543550491333, -0.5289106965065002, -0.7012616395950317, 1.870060682296753, 0.2156570702791214, -0.9453896880149841, 1.5118117332458496, -0.8240541815757751, -0.1995794177055359], [-0.046442121267318726, 0.49960771203041077, -1.0164427757263184, 0.26023560762405396, 0.14204570651054382, 0.5747970342636108, 0.2262447327375412, 0.5444238185882568, 0.03892834857106209, -1.4715548753738403, 0.8655656576156616, -1.2579511404037476, 0.4462779760360718, 0.3000680208206177, -1.3628506660461426, -0.03895392268896103, 1.1374765634536743, -0.7001190185546875, 0.531381368637085, 0.2340056449174881, 0.39737576246261597, -1.9269120693206787, 0.19486363232135773, 0.3154875338077545, -0.6498740315437317, 0.18261396884918213, -0.3737736642360687, 1.5678516626358032, 0.49455025792121887, 2.301884889602661, 0.35638228058815, -1.445081114768982, 0.7589585781097412, 1.9480113983154297, 0.44025713205337524, 2.106112480163574, 2.7986700534820557, -0.3111236095428467, -0.6519071459770203, -0.8161066770553589, -0.6410802602767944, 0.8021090626716614, -0.3179396688938141, 1.1849814653396606, -2.0701167583465576, 0.5070103406906128, -0.3332926332950592, -0.13856585323810577, -0.146039679646492, 1.7852599620819092, -0.92917400598526, -0.2935740649700165, 1.5946910381317139, -1.0352783203125, 0.8971198201179504, -1.7833060026168823, 0.7313882112503052, 1.0177714824676514, 0.4058762192726135, 0.9472557902336121, -1.086724042892456, -0.372516930103302, 1.0619550943374634, -1.3864160776138306, -1.486702799797058, -0.18848064541816711, 0.9598629474639893, -1.1202818155288696, 0.8254013061523438, 1.124529480934143, -1.4380204677581787, 0.3071911334991455, 1.6663583517074585, 0.8976930379867554, 1.0481849908828735, -0.5111175179481506, -2.7666285037994385, -0.5788179039955139, -0.1838410347700119, -1.801068902015686, 0.8077029585838318, 1.0293447971343994, 2.7352030277252197, -0.2932180166244507, -0.3742898106575012, -0.8665105700492859, 0.5167825222015381, -2.1261608600616455, -1.0146245956420898, 1.3095906972885132, 1.0166170597076416, 1.8993717432022095, -1.5234692096710205, -2.121469497680664, 0.4100147783756256, -0.22505606710910797, 0.906699538230896, 0.45825690031051636, 0.8064504861831665, -0.9438681602478027, 1.2887083292007446, 0.4207037687301636, -0.8373292088508606, -0.952497124671936, -0.07418368756771088, -0.09387606382369995, -1.1319389343261719, 0.6214069128036499, 0.18528282642364502, 0.8725873827934265, 0.674167275428772, 0.262933611869812, -0.3043930232524872, 1.7243447303771973, 0.7035760283470154, 1.8477176427841187, 0.8688035011291504, -0.29184412956237793, 0.20407196879386902, 0.7589537501335144, -0.7250698804855347, -0.06689715385437012, 2.2484049797058105, 0.2938442528247833, -0.168071910738945, -1.0634000301361084, -0.926850438117981, 0.21591752767562866], [0.07133454084396362, 0.7148353457450867, 0.30299025774002075, 2.6983063220977783, 0.6438822746276855, -1.046799659729004, 0.25475189089775085, 0.24567288160324097, 0.2691507935523987, 0.2844610810279846, 1.0703234672546387, -1.375123381614685, 1.6282954216003418, -0.7972655296325684, 0.3128851354122162, 0.32950475811958313, 1.2040374279022217, -0.12133114039897919, 0.4905872941017151, -0.7560923099517822, 0.11129488050937653, 1.9685744047164917, -1.7743806838989258, -2.2728729248046875, -1.9120452404022217, 1.0043801069259644, 1.0721880197525024, -2.0792970657348633, -0.8097328543663025, -1.0450106859207153, 2.618394374847412, -0.9196999669075012, -0.5191524624824524, -1.0829129219055176, -1.1972941160202026, -0.872637927532196, 0.5707590579986572, -1.8738042116165161, -0.7547021508216858, -0.41233792901039124, 1.5732885599136353, -0.8595330119132996, 0.8181495070457458, 1.2229093313217163, -1.591710090637207, -1.4415929317474365, -1.5064656734466553, -1.4236130714416504, -0.5975916981697083, 1.5680660009384155, 1.8024944067001343, 0.6568261981010437, -1.1048038005828857, -0.2591405510902405, 0.8377393484115601, -0.703408420085907, 1.173264980316162, 2.5562665462493896, -0.5607303380966187, 1.2539235353469849, -0.4320758581161499, -0.04825133830308914, 0.11529505252838135, 0.7511970400810242, 0.6944982409477234, -1.596347689628601, 0.9301213026046753, -0.5063067674636841, 0.15558815002441406, 2.0846235752105713, 0.1266651451587677, 0.22328771650791168, -0.005523177795112133, -0.1498127579689026, 3.267305374145508, -1.3631607294082642, -0.46051475405693054, -0.8755132555961609, 0.8240691423416138, 0.5364123582839966, 0.4618326723575592, -1.498364806175232, -0.32984867691993713, 1.2054239511489868, -0.344414621591568, -0.2863222658634186, -1.9374562501907349, 0.7487970590591431, 0.8741854429244995, 0.9065790176391602, -0.5318374037742615, 0.9767091870307922, 1.5151416063308716, -0.7512972950935364, -0.9200834035873413, -1.2575567960739136, 3.1427676677703857, -1.7458597421646118, 1.1957439184188843, 0.2705691158771515, 1.5482487678527832, -0.47392505407333374, 0.5943010449409485, -0.6751582622528076, -0.27338269352912903, -1.0799003839492798, -0.7264427542686462, 0.0823579952120781, 0.6670200228691101, -0.22157637774944305, 0.07005307823419571, 0.1289524883031845, 0.9182084202766418, 1.2281855344772339, -1.1961523294448853, 0.6427436470985413, -0.50533527135849, -0.24590091407299042, -0.161200612783432, -0.244585782289505, 0.2601703703403473, -0.5230402946472168, 1.421515941619873, -1.1372790336608887, -0.09751351177692413, 0.24275004863739014, -1.7284033298492432, -0.29611340165138245], [0.3415869176387787, 0.4098905920982361, -1.3228322267532349, -0.3828986883163452, -0.6398298740386963, 0.98599773645401, -2.3049094676971436, -1.0593352317810059, -1.4432461261749268, -0.39486610889434814, -2.0065083503723145, -0.790986180305481, 0.5884995460510254, 0.7728180885314941, -0.11433719098567963, 0.07852449268102646, -0.5436151027679443, 1.6052054166793823, -0.3171447515487671, 1.4791266918182373, 1.3937199115753174, -1.4772061109542847, -0.23299121856689453, 0.2806151807308197, -0.6834189295768738, 0.9045206904411316, 2.0090017318725586, -0.6035058498382568, 0.6337297558784485, 0.06460379809141159, 0.24948623776435852, 1.0129286050796509, -1.0290186405181885, -0.545438289642334, 0.4826964735984802, 1.424009084701538, -0.9312601089477539, -2.3754825592041016, -1.040137529373169, 0.4650886654853821, -0.011173692531883717, -0.5758437514305115, -0.5860098004341125, -0.10959812253713608, 0.6843698620796204, -0.41180846095085144, 0.8855896592140198, 0.425368070602417, -0.6208540201187134, -0.0036329329013824463, -1.8793110847473145, 0.355538010597229, -0.32786446809768677, 0.042808420956134796, 0.8423029184341431, -0.3309938609600067, 0.5733328461647034, 0.1413014978170395, 1.9757821559906006, -0.3780826926231384, 1.388608694076538, -0.16344447433948517, -2.030566453933716, -0.11000113189220428, 0.025460122153162956, -0.7765893340110779, -1.403263807296753, -1.448992133140564, -0.6759817600250244, -1.208996057510376, 0.9754615426063538, -0.08400052785873413, 0.6872710585594177, -0.08217325806617737, -1.8924905061721802, 1.256866455078125, 0.41644009947776794, 0.18405520915985107, 0.2949616611003876, 0.9611855745315552, -0.10228385031223297, -1.7392563819885254, -0.5908181071281433, -1.7019628286361694, -0.5222721099853516, 0.08773031830787659, 0.34437263011932373, 0.3071514666080475, 0.5090596079826355, 1.3557323217391968, -0.05571447312831879, 0.26390793919563293, -1.6638439893722534, 1.3190940618515015, -0.1461753100156784, 0.06309062242507935, 0.10162671655416489, -0.6859514713287354, -1.086370825767517, -0.753956139087677, -2.6292402744293213, 2.215956211090088, -1.658315658569336, 0.3562425374984741, -1.1405614614486694, -0.298005074262619, 0.0800415500998497, 0.9069143533706665, 0.14026190340518951, -0.30947962403297424, -0.0015283634420484304, -0.3474198281764984, -2.2123591899871826, 1.641648530960083, 0.20731930434703827, -1.0525915622711182, 0.043309617787599564, -0.1451541930437088, -1.0475164651870728, 1.3578190803527832, -0.9279645681381226, -1.1706902980804443, 0.4096982479095459, -0.425187885761261, 1.9290097951889038, -0.6180810928344727, -0.3188857138156891, -0.34557148814201355], [1.4966797828674316, 0.7073333859443665, -0.5444762110710144, 0.4464268386363983, -0.766430139541626, 1.2346705198287964, 0.3758060336112976, -0.21512600779533386, -0.38094866275787354, 1.2099456787109375, 2.496716260910034, 0.06959816068410873, -2.307718276977539, -1.965876579284668, -0.2372513711452484, -0.7662196755409241, -1.2120556831359863, -0.06520646810531616, 0.2674642503261566, 0.5060209631919861, -0.8603900074958801, -0.8946056962013245, -0.0030153554398566484, -2.1358542442321777, -0.6061826348304749, -0.12509000301361084, -0.20602937042713165, -0.5997395515441895, -0.18155419826507568, -0.7368314862251282, 0.7610746026039124, -1.6856375932693481, -1.8820984363555908, -0.2988300323486328, -0.9525573253631592, 1.5839828252792358, 0.27862784266471863, -1.5327221155166626, -1.2981938123703003, -0.7468056082725525, -0.5562886595726013, -2.3339719772338867, 0.34235623478889465, -0.6391807794570923, -0.44792574644088745, -0.36671188473701477, -0.8119676113128662, -1.7196049690246582, 2.080984354019165, -1.6673343181610107, 0.06478450447320938, -0.0009864663006737828, 0.23283950984477997, -1.2802464962005615, 0.03562844917178154, -0.1632324606180191, -1.0513256788253784, -0.003958375658839941, 0.21032387018203735, -0.052201926708221436, 0.9012187719345093, -0.23182089626789093, 0.23899538815021515, 0.6196016669273376, -0.18680155277252197, 1.0550339221954346, -1.3226398229599, -0.9948657751083374, 1.5591864585876465, -0.22976624965667725, 0.69426029920578, -0.38833358883857727, -0.6811390519142151, 0.27232834696769714, 0.5953276753425598, -0.6514990925788879, -0.6657447218894958, -1.6724644899368286, 0.031348906457424164, 0.4647347033023834, 0.2267165184020996, -1.698360562324524, -0.6669340133666992, 0.700044572353363, -3.074126720428467, 0.7366824746131897, -1.3016456365585327, 0.8821328282356262, -1.4731450080871582, 0.06804698705673218, -0.39452096819877625, 0.6066170334815979, 0.034868452697992325, -1.7342917919158936, -0.39270997047424316, -1.0585957765579224, 1.0134994983673096, 1.5521057844161987, 2.8033254146575928, -0.9259083867073059, -0.5234156250953674, 0.2988153398036957, 0.11263211071491241, -1.485242247581482, -0.8884425163269043, -1.3078831434249878, -1.6670286655426025, -1.428885579109192, 0.2936629354953766, 0.7443209290504456, -0.46927350759506226, 1.5361114740371704, 1.5795270204544067, 0.20856209099292755, 0.4961397051811218, -1.8422728776931763, -0.21397764980793, -1.3786441087722778, -0.898253321647644, -0.7042690515518188, 0.9050480127334595, 0.16829226911067963, 0.37338435649871826, 1.0153156518936157, -0.4059544801712036, 1.2396979331970215, 1.2961174249649048, 1.2019543647766113], [-0.5448349714279175, -0.6538594365119934, 0.10636527836322784, -0.06705253571271896, 0.13006828725337982, 0.1301904022693634, 0.5905560851097107, -1.538641095161438, -0.6885644197463989, -1.2374238967895508, -0.3605017066001892, -1.2792719602584839, -0.5968108177185059, -1.3810325860977173, -0.6613008379936218, -1.6336978673934937, 0.04340630769729614, -0.5372775197029114, -0.14082004129886627, -0.4321914315223694, 0.05165732651948929, 1.845462441444397, -2.3429501056671143, -0.14028871059417725, -0.2839616537094116, 2.560360908508301, -0.6609848737716675, -0.08303939551115036, -1.2328057289123535, -1.5790133476257324, -1.5573227405548096, -1.167555809020996, 0.9163313508033752, 0.8388030529022217, -1.8099387884140015, 0.3712669909000397, -0.9496424794197083, -0.7950069308280945, 2.638254404067993, -1.9822415113449097, -1.688664436340332, 1.1776444911956787, 0.25015848875045776, -0.5950843691825867, -1.2127245664596558, -1.4387871026992798, 0.48468640446662903, 0.3547612130641937, -1.081121802330017, 1.852338194847107, 1.7509816884994507, -1.0914866924285889, 0.633405327796936, 2.294842004776001, 0.13764327764511108, 0.8493794202804565, 0.6315383315086365, -0.7882592678070068, -0.8836575746536255, 0.15192270278930664, 1.2484511137008667, -0.6645915508270264, -0.008799061179161072, -0.0945013016462326, 1.480391263961792, -1.7088323831558228, 1.046519160270691, -0.5420107245445251, 0.1857251226902008, 0.7044722437858582, -0.07630116492509842, -1.1792192459106445, 0.2046826183795929, 2.5013086795806885, -0.13421712815761566, -1.043571949005127, 2.4589016437530518, -0.3585955500602722, -0.7005820274353027, -1.0315923690795898, 0.362748920917511, -1.5153270959854126, -1.618046760559082, -0.34299132227897644, 0.6000509858131409, -0.7249042391777039, -1.4334992170333862, 1.0573229789733887, -2.1432077884674072, 0.1111038476228714, -0.16008202731609344, 0.37803032994270325, -2.201932907104492, -0.45881912112236023, 0.9070983529090881, -1.336633324623108, 1.7256592512130737, -0.26274392008781433, 0.28612688183784485, -1.0095791816711426, 0.3745553493499756, -2.2106528282165527, -0.8806321620941162, 1.1579087972640991, -1.0500630140304565, -0.6832361817359924, -0.7422600984573364, -2.3965070247650146, -0.2215854823589325, -0.5292284488677979, 0.02414042130112648, -0.3320809006690979, -0.7457497715950012, -0.5845398306846619, 1.3654457330703735, 1.0697705745697021, -0.6291271448135376, -0.23116420209407806, 0.7910046577453613, 0.7982575297355652, -0.566076934337616, 0.9856944680213928, -0.860139012336731, -0.3095357120037079, 1.2554229497909546, 0.2965185344219208, 0.6575779318809509, 0.06364902853965759], [2.4058048725128174, -1.3902108669281006, 1.8958390951156616, 0.7024545073509216, -1.126832127571106, -0.2966134548187256, 0.16796518862247467, 0.8057254552841187, -1.2157450914382935, 0.41616857051849365, -0.5362061858177185, -1.2104493379592896, 2.0989186763763428, -1.5665974617004395, 0.4160737097263336, 0.34445926547050476, 0.20322369039058685, -1.1933693885803223, 2.542703151702881, 0.4092884361743927, 0.783496618270874, 1.5177175998687744, -0.7341756820678711, -0.3852780759334564, -1.6329052448272705, 1.1247062683105469, -0.860927939414978, 0.38220351934432983, -1.0197453498840332, -1.0879089832305908, -0.33050477504730225, -0.47107481956481934, 0.9961860179901123, -0.7906244397163391, -0.8213198781013489, 0.6303896903991699, 2.7459516525268555, 0.36211860179901123, 1.417954921722412, 0.06590939313173294, -1.4077661037445068, 0.5026443004608154, -0.660822868347168, -1.1019630432128906, 0.7712182402610779, 1.036072850227356, -0.6232975721359253, 0.510631263256073, -0.0050443983636796474, -0.20200596749782562, -0.6846966743469238, -0.929283618927002, -0.26846078038215637, -0.5873982310295105, -2.10774302482605, -0.41585060954093933, 1.6665669679641724, -0.6492937207221985, 1.4543101787567139, 0.7990175485610962, -0.8193761110305786, -1.1137418746948242, 0.9213749766349792, 0.11822991818189621, 0.6037676334381104, 1.0437406301498413, 0.9473679661750793, 0.193405419588089, -0.800128698348999, -1.1532405614852905, -1.1680606603622437, -0.11689800024032593, -0.09297595173120499, -0.6178513765335083, -0.44855478405952454, -1.2734503746032715, 0.6983844637870789, -2.12036395072937, 0.4750642776489258, 0.9240777492523193, 0.16462449729442596, -0.025047635659575462, -0.07511328905820847, -1.0716187953948975, -1.1847504377365112, 0.14656691253185272, -1.1197408437728882, -1.021487832069397, -1.1088680028915405, -1.3403668403625488, 1.5056349039077759, 0.7405686378479004, -0.7428438067436218, -0.2620103359222412, 1.8450639247894287, 0.6302312612533569, 1.1793467998504639, -0.31689774990081787, 0.10501818358898163, -0.8604698777198792, -0.8335145115852356, 1.516767144203186, -0.40902698040008545, -0.3913189470767975, 0.003071303479373455, -0.023307843133807182, -0.3308752775192261, -1.250303030014038, 0.13005411624908447, 0.043299343436956406, 1.3519010543823242, -1.4915918111801147, -0.8144456744194031, -0.13715840876102448, 0.0863996222615242, -0.9866355061531067, -0.969819962978363, -0.627956211566925, -0.47606149315834045, 2.0120882987976074, 1.136020302772522, -0.16768574714660645, 1.1903784275054932, -1.2199848890304565, 0.15445569157600403, -0.009964637458324432, 0.31273287534713745, 0.11379015445709229], [0.5743690133094788, -0.045354537665843964, 0.3174956440925598, 1.347922921180725, -0.701888382434845, 0.8892487287521362, 0.18731144070625305, -1.0914065837860107, -1.8102271556854248, -0.7336317300796509, -0.30742573738098145, 0.9399821758270264, -0.3223530948162079, -1.0191888809204102, -0.49009644985198975, -1.002051830291748, -0.5197510123252869, 0.8464227318763733, 0.6514694690704346, 0.028718795627355576, -0.1320337951183319, -0.20015858113765717, -1.0932425260543823, 0.42949941754341125, -1.140347957611084, 1.092326283454895, -0.26297813653945923, 2.2337229251861572, -2.7061381340026855, -0.03744252398610115, 1.3643181324005127, 0.4631238281726837, -0.5318794846534729, -1.4167888164520264, 0.6926863193511963, 1.0129578113555908, -0.7011533379554749, 1.7876718044281006, -0.3881591260433197, -1.6379003524780273, -0.6566240787506104, 0.3041176199913025, 0.3579637110233307, -1.0022615194320679, 0.6426138281822205, -0.11353416740894318, -0.6764313578605652, -1.5862398147583008, 1.055436372756958, -0.5441084504127502, -1.1643871068954468, -0.2838454246520996, 0.8687134385108948, -0.24707850813865662, -0.5186542272567749, 0.6268989443778992, -2.2661149501800537, -0.26606783270835876, 0.7786635756492615, -1.1359314918518066, -0.07684430480003357, -0.17415273189544678, 0.4954302906990051, -0.2508939504623413, 1.1027116775512695, 1.063392996788025, -1.1537288427352905, -0.2519280016422272, 0.4326845705509186, -0.24109818041324615, -0.33888402581214905, 0.7468847632408142, -0.7909058332443237, -1.7501158714294434, 0.9219852685928345, -0.2477012425661087, -1.2721308469772339, -1.2824196815490723, 0.9231288433074951, 0.13264180719852448, 0.10092294216156006, 1.230846881866455, 2.066342830657959, -0.13771356642246246, 1.1687345504760742, 0.9764886498451233, -1.0446141958236694, -0.9222105145454407, -0.1625710427761078, 0.8162770867347717, -0.24977098405361176, 0.9643917083740234, -1.7332491874694824, 0.6804219484329224, -0.2418777197599411, -0.06191820651292801, 0.5298125147819519, 1.2783771753311157, -0.3615409731864929, -1.1703368425369263, 0.8248857259750366, -0.9159292578697205, 2.548825979232788, 0.8829517364501953, -1.2131645679473877, -1.2101619243621826, 1.8647629022598267, 0.9551164507865906, 0.6580321192741394, 1.3758351802825928, 0.7610732913017273, -0.06110682338476181, -1.3989200592041016, 1.17206871509552, -0.3394853472709656, 0.5919039249420166, 0.5630564093589783, -0.6099494099617004, -0.3315598964691162, -0.16089962422847748, -2.1949827671051025, 1.3511303663253784, 0.29037052392959595, -1.1855863332748413, 0.6192107796669006, -1.8575845956802368, -1.2856284379959106, 0.5031406879425049], [-1.2334742546081543, 0.41218847036361694, -0.26513928174972534, -0.635280191898346, -0.016397884115576744, -0.5176244378089905, 1.7953455448150635, -1.0779069662094116, -0.5776312351226807, -0.2485867589712143, 0.6485860347747803, 0.5543619394302368, 0.8095405697822571, 0.48485347628593445, 0.9142516255378723, 1.1816591024398804, 0.018114512786269188, -0.46271032094955444, -0.6023079752922058, -0.6899632811546326, 1.3270456790924072, -1.6810790300369263, 0.007703823037445545, -0.3045225143432617, 0.6271883249282837, 0.9169082641601562, 0.6089518666267395, 0.38796019554138184, -2.2606334686279297, 1.0720889568328857, -0.5759401917457581, 0.23882216215133667, 0.48921456933021545, 0.7237727046012878, -0.9121465682983398, 0.22792385518550873, -0.8747768402099609, 0.6193016171455383, -0.3732239902019501, -0.376904159784317, 0.3585066795349121, 1.1426706314086914, 0.21960538625717163, -0.18618600070476532, 0.599839985370636, 0.10225387662649155, 0.031620774418115616, 0.5985395312309265, 0.9210568070411682, -1.0328108072280884, -1.2405688762664795, 2.4402036666870117, 0.5971885323524475, 0.7264350652694702, 1.3735522031784058, -0.43213772773742676, -0.8343530297279358, 1.6096625328063965, -1.4492237567901611, 0.09721574187278748, 0.38244473934173584, -1.1542705297470093, -1.7326246500015259, 0.23070314526557922, -0.7300545573234558, 1.7175143957138062, 1.2670539617538452, -1.7768614292144775, 0.6330676078796387, -1.4930431842803955, -0.03326817974448204, -1.2743425369262695, 0.4351250231266022, -0.45321252942085266, -1.0679125785827637, -0.5029082894325256, 0.0044005075469613075, -0.17768089473247528, 0.12129126489162445, -0.8243196606636047, 0.12228765338659286, -0.7517713308334351, -1.0822579860687256, 0.16547268629074097, -0.9586005806922913, 0.19909313321113586, -0.7560180425643921, 0.9469872117042542, -0.09391510486602783, 0.6607877612113953, -0.5142204761505127, 1.7467676401138306, -0.17106731235980988, 0.2845803499221802, 0.3110663890838623, -0.33655378222465515, -0.6470866799354553, -0.18533611297607422, 0.6754282116889954, -0.9639864563941956, 0.7422367334365845, 0.6005297899246216, -1.585045576095581, 0.48649755120277405, 0.5095882415771484, -0.23137259483337402, 0.09910878539085388, 0.4383106529712677, -0.3901330828666687, 0.30734533071517944, 0.23837167024612427, -0.45034199953079224, 0.29271289706230164, -1.2862211465835571, 1.5311386585235596, 0.8141292929649353, 1.115992546081543, -0.009458374232053757, -0.3454938530921936, -1.0617296695709229, 0.32084983587265015, -1.8289170265197754, 2.1115128993988037, 0.020388778299093246, 0.4873046875, 0.8763259649276733, 0.8713140487670898, 2.0329010486602783], [-0.1434638947248459, 0.03478558734059334, 0.023926621302962303, -2.6107027530670166, -0.7131026387214661, -0.9511865377426147, 0.6347734332084656, -0.6422547101974487, -1.0229942798614502, 2.0261077880859375, -2.0359935760498047, -1.9701515436172485, -0.08627495914697647, 0.6244564652442932, 0.9512367248535156, -0.8669996857643127, -0.04830057919025421, -0.1395580768585205, -0.4089605212211609, 0.48658183217048645, -0.08787237107753754, 0.26021531224250793, 1.8974685668945312, 0.08910571038722992, 0.0006454512476921082, -0.21526727080345154, 0.20035246014595032, -0.18025735020637512, -0.6651676297187805, 0.19542039930820465, 1.965798258781433, 0.216425359249115, 0.062446944415569305, 0.005113063845783472, 0.2860840857028961, -1.4670318365097046, 1.689595103263855, -0.8769784569740295, -1.228563904762268, -0.01914862170815468, -0.9902414083480835, 0.015067395754158497, -0.6922195553779602, -0.21278096735477448, 0.3360278010368347, -1.2344800233840942, -1.260949969291687, -1.135233998298645, -0.11463613063097, -0.08960755169391632, -1.1543986797332764, 2.120271682739258, -0.09369337558746338, 0.4105817973613739, 1.1419882774353027, -2.120985746383667, -0.8871040940284729, -0.8709208369255066, 1.509867548942566, -0.4268661439418793, 1.9008829593658447, 0.0006482956232503057, -1.1442468166351318, -0.637718677520752, -0.21452932059764862, 0.7882042527198792, 0.7176070213317871, 0.05417439714074135, -1.6229875087738037, -0.47490331530570984, -1.0739156007766724, -1.5924137830734253, 1.2425259351730347, 2.0578858852386475, -0.5025575160980225, 1.4508386850357056, 0.8071937561035156, -0.26823922991752625, -2.26816725730896, 0.4340798556804657, -1.1796977519989014, 1.289246916770935, 1.0550172328948975, 0.19607266783714294, -2.5219600200653076, 0.379743754863739, -0.7475796341896057, -0.0726935863494873, -0.42572519183158875, 0.1425880789756775, -0.9264352321624756, -1.463135838508606, 2.0315897464752197, 1.2588168382644653, -1.6653281450271606, -0.4850917160511017, -0.8507136702537537, -0.860331654548645, -0.2364986687898636, 0.915702223777771, -0.9067794680595398, -0.4024834930896759, 0.8179227709770203, -1.0150089263916016, 0.08572877198457718, -1.4933712482452393, -0.44178006052970886, -0.7858129143714905, 0.7372031807899475, -2.050598382949829, -0.3785035312175751, 0.7491431832313538, 0.5928134918212891, 0.14363351464271545, -1.0500929355621338, -0.4183628261089325, -0.029216498136520386, -1.025417685508728, 0.41994717717170715, -0.12483203411102295, 0.31835636496543884, 0.2527090907096863, -0.5283761620521545, 0.9495267868041992, -0.6893914937973022, 1.7666245698928833, 0.42842742800712585, -0.16942669451236725], [0.32559284567832947, -3.2069926261901855, 0.7586715221405029, -1.6808457374572754, 1.3749651908874512, 0.14197133481502533, -1.3335187435150146, 0.7022911310195923, -0.10840439796447754, 1.6132750511169434, 0.434334397315979, 0.5254630446434021, 0.6905080080032349, -2.1754229068756104, -1.2043777704238892, -0.9072308540344238, 0.3766094148159027, -0.6796517372131348, 0.7385916709899902, -0.6053966283798218, 2.1243860721588135, 0.7719448804855347, -1.5046454668045044, 0.6434707641601562, -0.1549271047115326, 1.0720593929290771, 1.2678732872009277, 0.384977251291275, 1.0878734588623047, 0.319172203540802, 0.733086884021759, 1.3635891675949097, 0.6735236048698425, -0.5876824855804443, 0.7559310793876648, -0.5409736037254333, 0.4738020598888397, 0.8865019083023071, -0.7442786693572998, -1.0904455184936523, 2.2069334983825684, -0.3605547547340393, 0.6261866688728333, 1.2648088932037354, 1.2130086421966553, -0.571685791015625, 1.8290108442306519, 0.8194690346717834, -0.31402185559272766, 0.9515164494514465, 0.07923369109630585, 1.2101083993911743, 0.8210982084274292, -0.6174358129501343, 0.8258146047592163, -0.9502490162849426, 0.6195757985115051, 0.4480982720851898, 0.47281137108802795, -0.06683260202407837, 1.3921560049057007, -0.5083502531051636, -1.2936981916427612, -0.6507993340492249, -2.472191333770752, -0.12866443395614624, -0.40004798769950867, 1.556058406829834, 0.17795222997665405, -0.11278601735830307, -1.7198045253753662, -2.1919591426849365, -0.022551847621798515, 1.0474389791488647, -0.5225539803504944, 0.7633996605873108, 1.5621446371078491, -1.44564950466156, -0.15945635735988617, -0.7242334485054016, -0.3433801233768463, 0.3440159559249878, 0.46164578199386597, -0.028916826471686363, -1.1367127895355225, -1.7564083337783813, 2.326200246810913, 2.2217657566070557, 1.608199954032898, 0.05480891466140747, 1.865299105644226, -0.38003093004226685, -0.8688507080078125, -1.5538502931594849, -1.9392116069793701, 1.015571117401123, -0.5367450714111328, 0.24276216328144073, -0.050095707178115845, -0.8472326993942261, -0.23110677301883698, 0.3286229968070984, -0.24859902262687683, -0.45872434973716736, 0.28583770990371704, -0.3287721574306488, -0.36393508315086365, 1.473534345626831, -0.46205654740333557, -1.7218810319900513, -0.6551443934440613, 2.638230562210083, -0.1935388296842575, 0.25034192204475403, 0.3707800805568695, 0.9494815468788147, -0.7576781511306763, -1.7422842979431152, -0.300674706697464, 0.11963842809200287, 1.467484712600708, -0.546644389629364, -1.7993301153182983, -2.191836357116699, 0.14054076373577118, -0.8145001530647278, 0.07336178421974182, -1.0518956184387207], [1.3259540796279907, -0.46886304020881653, 0.9963352680206299, -1.4948554039001465, 0.20053182542324066, -0.2862773835659027, 0.19120654463768005, 0.3636232316493988, 0.6611040830612183, 1.1020828485488892, 1.0964852571487427, 0.569385826587677, -1.1256654262542725, 0.15119026601314545, -0.7034108638763428, -1.8953250646591187, -0.30039846897125244, 0.8108792304992676, -0.9952106475830078, -0.6961703896522522, -0.04476963356137276, -1.8017522096633911, -1.7727434635162354, -1.0460119247436523, -1.4736402034759521, 1.2195042371749878, 3.220217227935791, 0.5137662291526794, -0.30453985929489136, 0.08452893048524857, -0.30369606614112854, -0.37886714935302734, 0.03511252626776695, -1.8280543088912964, 0.31869620084762573, 1.238926887512207, -0.5879043936729431, 0.28200361132621765, -0.15784944593906403, 2.4952712059020996, 0.8915639519691467, 1.336662769317627, 0.4459436535835266, -1.8708686828613281, -0.9364416003227234, 0.575901985168457, -0.41791942715644836, -0.7808595895767212, -0.7992053031921387, -2.908905267715454, 1.299010992050171, 0.29352420568466187, 0.062484242022037506, -1.8526191711425781, -0.14757953584194183, -0.20516222715377808, -0.6868796348571777, -1.7277640104293823, -0.07821132987737656, -1.103277564048767, -0.038167789578437805, 2.1172335147857666, -1.5710501670837402, 1.5400029420852661, -1.0911616086959839, -0.030433334410190582, 0.8281134963035583, -1.496177077293396, 0.19514283537864685, 1.0855185985565186, -0.4295819401741028, 0.42165112495422363, -0.588356614112854, -0.9606025815010071, -0.8920746445655823, -0.12917791306972504, -0.19862762093544006, -0.048264745622873306, 0.3269280791282654, 1.1999599933624268, -1.3769176006317139, -0.9190211296081543, -0.4106813371181488, -0.5224263668060303, -1.8945022821426392, -0.25919872522354126, -0.07321371883153915, 0.6248391270637512, -0.4184967875480652, 0.7717796564102173, -0.27964961528778076, -0.894193708896637, 1.7666994333267212, 0.09497860819101334, 0.08867047727108002, -0.5389329791069031, -0.4587211310863495, 0.6454734206199646, -0.02081906981766224, 0.9380218386650085, -1.0218942165374756, 0.2719208002090454, 2.5969290733337402, -0.12377645820379257, 0.673797607421875, 0.018117351457476616, 1.3651100397109985, 1.6069060564041138, -0.3144601881504059, -1.320393443107605, 0.48096904158592224, -0.017439749091863632, -0.5410400629043579, 0.686305582523346, 0.5736666321754456, 0.33898162841796875, -0.36152225732803345, 2.715886354446411, -0.03734251856803894, 0.34701448678970337, -0.0554235614836216, 0.6438786387443542, -1.624765396118164, -0.7081607580184937, 0.7030084133148193, 0.8171719908714294, 1.5449187755584717, -0.3601996600627899], [-0.3422241508960724, -1.0484273433685303, 0.5386779308319092, -0.35804229974746704, 0.9158275723457336, 0.506883978843689, 1.1386185884475708, -0.7965595722198486, -0.030974816530942917, 0.09695062786340714, 1.7001152038574219, -0.9397687911987305, -0.19949810206890106, 0.40707364678382874, -1.2872405052185059, -0.5188665390014648, 0.21514999866485596, 0.45770782232284546, 0.20358668267726898, -0.05198938399553299, 0.8318654894828796, -0.020268399268388748, -0.6469636559486389, -0.10651684552431107, 1.5443648099899292, -0.8000726103782654, 0.9533582925796509, 1.6627109050750732, 1.0962811708450317, 0.8414578437805176, -1.70476233959198, 0.6337879300117493, -0.9028641581535339, -0.47200068831443787, 0.7897980213165283, 0.012413705699145794, -0.37964192032814026, -0.42528194189071655, -0.9536346793174744, -0.04828221723437309, 0.25319647789001465, 0.021582456305623055, 1.4320487976074219, -0.27289631962776184, 0.7638423442840576, -0.45433107018470764, 0.32777178287506104, 0.16689859330654144, -0.1912619173526764, 0.3778730630874634, -0.31827086210250854, -0.973351001739502, 0.41995513439178467, 1.2506998777389526, -0.6020958423614502, -0.9610090851783752, -1.2214164733886719, -0.5349230766296387, -1.4184490442276, 0.13093534111976624, -0.560685396194458, 0.8069223165512085, -0.2930733561515808, 0.47867026925086975, 0.49639892578125, -0.19479279220104218, 0.5582799911499023, 0.9540094137191772, 0.11684253066778183, 0.15480786561965942, -0.32748943567276, -0.557400643825531, -0.3056713342666626, -1.4983625411987305, 2.3043253421783447, -0.1337301880121231, 0.7857657074928284, 0.42506715655326843, 1.4155336618423462, 0.3199334740638733, -0.550422728061676, -0.7042322754859924, -0.9094478487968445, 0.7989615797996521, -1.802517294883728, -1.3324639797210693, -1.51375412940979, 0.6237918138504028, -0.6121072769165039, 0.28554806113243103, -0.2484472543001175, -0.19398511946201324, 0.07437090575695038, -0.4676527678966522, 0.027975112199783325, 1.600877285003662, -1.4005467891693115, -0.5624998807907104, -1.3851845264434814, -0.6732118725776672, 0.8408776521682739, -1.0550286769866943, -0.26640745997428894, -0.8359549045562744, 0.01364278793334961, 0.4518458843231201, 0.34535783529281616, 0.15374483168125153, -1.0099201202392578, 0.10095579922199249, -0.259406179189682, 1.4464991092681885, -0.2457250952720642, 1.4778193235397339, -1.0691478252410889, 0.034165482968091965, 0.38227424025535583, -0.3517643213272095, -0.007668295409530401, -0.3608549237251282, -0.004508520010858774, -0.655418872833252, 0.7754258513450623, -2.091057538986206, 0.7163437008857727, 2.9123003482818604, -0.18717777729034424, -2.0212106704711914], [0.1295255422592163, 0.025727728381752968, 1.4999046325683594, -0.9172921776771545, -0.49647223949432373, 1.5103594064712524, -0.5688257217407227, -0.33917027711868286, -0.5425384044647217, -0.4253283441066742, 0.14590047299861908, -0.5338345170021057, -1.1002647876739502, 1.2752350568771362, 0.019029930233955383, 1.0491466522216797, -0.45910581946372986, 0.2555931508541107, 0.47008654475212097, 1.5344171524047852, -0.42078980803489685, 0.25392428040504456, 1.5903668403625488, 0.37671834230422974, 0.4724943935871124, -0.6326619386672974, -1.2679344415664673, -0.03307213634252548, -1.6617454290390015, 0.6995481252670288, -0.6435511112213135, 2.03433895111084, -0.6471961140632629, -0.17305539548397064, -0.6257126927375793, 1.9472461938858032, -0.4475027024745941, -1.2192553281784058, -1.9895381927490234, 1.978202223777771, -0.4446767568588257, -1.978864312171936, -0.681441605091095, -0.38052651286125183, -0.3165026009082794, 0.10622365772724152, -0.009019673801958561, 0.9822241067886353, -0.8949317932128906, 0.056376419961452484, 0.4116869270801544, -0.1779167354106903, 1.7519488334655762, -1.612200379371643, -0.6853870749473572, 0.5823386311531067, -1.1242923736572266, 1.759131669998169, 0.17072412371635437, 1.1296052932739258, 1.0874156951904297, -0.1968151032924652, -0.3618282973766327, 0.780326247215271, 1.2536829710006714, -0.5744118094444275, -0.7090657353401184, -1.6313532590866089, 0.7043729424476624, 0.718387246131897, 0.39646485447883606, 0.3369874358177185, 0.9549617171287537, -0.1137109249830246, 1.2454699277877808, -1.9958343505859375, 0.24856413900852203, -0.4118204712867737, 0.11431146413087845, -0.06342137604951859, 0.6712125539779663, 0.02380428835749626, -1.004030466079712, -2.1739165782928467, -0.3837897777557373, -0.7464349269866943, 0.6689980626106262, 0.2183867245912552, -0.023151561617851257, 0.7510741353034973, -0.5098577737808228, 0.1923995465040207, 0.22205306589603424, 0.7330537438392639, -1.0352199077606201, 0.5240283012390137, 1.0847249031066895, -0.6449510455131531, 0.5104984045028687, 0.638809323310852, 0.6949374079704285, 0.10016556084156036, -0.7432932257652283, -0.22387148439884186, -0.6355576515197754, 0.33061841130256653, 0.39408811926841736, 0.9669398665428162, 0.19098900258541107, -0.5643817186355591, -1.9617058038711548, 1.0164028406143188, 0.3297539949417114, 1.135435938835144, 0.18097954988479614, 0.2097703516483307, 0.7876915335655212, 1.174055576324463, -0.1712811440229416, -0.7739726305007935, 0.7260609865188599, -0.5373103022575378, -2.939800500869751, -1.0590883493423462, 1.530755877494812, 0.7471773028373718, -0.00762362452223897, -0.5934205055236816], [2.324793815612793, 1.2349984645843506, -0.8088658452033997, 1.5298783779144287, 0.7184552550315857, 0.6915997266769409, 1.9485613107681274, 0.9158158302307129, 1.7013856172561646, -1.653555989265442, -1.123604416847229, 1.2484647035598755, 0.1136171817779541, -0.14909851551055908, -1.4041407108306885, 0.8759653568267822, -0.4386990964412689, -0.41716355085372925, 1.096096396446228, -0.37374037504196167, -0.6118348240852356, 1.6672781705856323, -0.6780115962028503, 0.49066364765167236, 0.6076313257217407, -1.5206037759780884, 1.5701055526733398, 0.5824025273323059, -0.6696657538414001, -0.12070334702730179, -0.7745432257652283, -1.5858683586120605, 0.49046066403388977, -0.5659847259521484, -0.4239670932292938, -1.2349778413772583, -0.46770697832107544, 0.033332470804452896, 0.20260943472385406, -0.3602589964866638, -0.1458231657743454, -0.4463059902191162, -1.5109238624572754, 1.744886040687561, 1.3075025081634521, 0.056381601840257645, 0.008415058255195618, -0.3031042814254761, -0.6423845291137695, -1.1953046321868896, -0.31339383125305176, 0.2634795606136322, 0.7327013611793518, 0.8888078331947327, -0.9399215579032898, -0.5770494937896729, 0.6292486190795898, -0.1659337431192398, -0.3552817702293396, -1.9555633068084717, 0.9660831689834595, 1.8555073738098145, -1.260047197341919, -1.246465802192688, 1.3593493700027466, -0.813503086566925, 0.023320313543081284, -0.19624172151088715, -0.4992614984512329, -0.8178611397743225, 0.20492658019065857, 0.2376101315021515, -0.7559704184532166, -0.9040763974189758, 1.6674859523773193, 0.5010987520217896, 2.3880090713500977, 0.3351198732852936, 0.4739648103713989, -0.10180950909852982, -0.7190423607826233, -0.11669560521841049, 1.5395994186401367, -0.6901903748512268, 1.2321090698242188, 0.857822597026825, -1.448258876800537, 0.8279076814651489, -0.7986900806427002, 0.28588539361953735, 1.1698429584503174, -0.7143489122390747, -1.291625738143921, 1.1195393800735474, -0.9695481061935425, -0.9116458892822266, -1.2416712045669556, 0.633928656578064, -0.8299103379249573, 1.0706268548965454, -0.04148437827825546, 0.0666511207818985, -1.296661138534546, 0.09362450987100601, 0.516510546207428, 0.9745396971702576, -1.2296086549758911, -0.9795386791229248, -0.8981298208236694, 0.14644424617290497, 0.03311556205153465, 1.3581370115280151, -0.6279401779174805, -1.6218899488449097, 1.9054256677627563, -1.4274089336395264, 0.18885955214500427, -1.9180033206939697, -0.33605512976646423, 0.5494349598884583, 1.0928295850753784, -1.4054325819015503, -1.8949496746063232, 0.2556266486644745, 0.9549838900566101, -0.957542896270752, 1.6365606784820557, -0.7698417901992798], [0.33298707008361816, -0.39756718277931213, 1.8008917570114136, 1.2438087463378906, -0.6453825235366821, 0.08847230672836304, 0.137694850564003, 0.17790406942367554, 0.06360190361738205, 2.3459815979003906, 1.1902512311935425, -0.5851159691810608, -0.7954574823379517, -1.3547683954238892, 0.04337245598435402, -0.7759516835212708, -0.5590149164199829, 0.283080518245697, 0.04516022279858589, -0.1497667282819748, -0.0600602850317955, -0.13704612851142883, 0.9618845582008362, 0.3927536904811859, -0.17433564364910126, -1.0131961107254028, -1.2020477056503296, 0.8391923904418945, -1.1875663995742798, -1.60509192943573, -1.468446135520935, -2.146143674850464, 0.0407208688557148, -0.8114156126976013, -0.7244293689727783, -1.4475011825561523, -0.1154497042298317, 0.00673707528039813, -0.1934041678905487, 1.1621153354644775, -0.08948791027069092, 0.3129936158657074, -0.11975828558206558, -0.15534426271915436, 0.012651621364057064, 0.6835159659385681, -0.48513272404670715, 0.6512572765350342, -0.3125758767127991, -0.067349374294281, 0.29866909980773926, 0.31652355194091797, 0.5432313680648804, -1.05205237865448, 0.9024486541748047, -0.13973622024059296, 0.3428837060928345, -0.700950562953949, 1.8381303548812866, -0.22328965365886688, 1.2490185499191284, 0.7048457860946655, 0.8466477990150452, -0.23771575093269348, 0.3647960126399994, 0.973812460899353, 0.651777982711792, 0.010137946344912052, -0.025091974064707756, -0.035555317997932434, -0.34721019864082336, -0.8547174334526062, 0.20052726566791534, -0.027943748980760574, 0.5624635815620422, 1.6863476037979126, 0.809086263179779, 0.304558664560318, 0.11744380742311478, 0.1231146976351738, 1.0916311740875244, 0.07554159313440323, 0.6559461355209351, -0.022115785628557205, 0.41579848527908325, 0.19027210772037506, -0.4890539348125458, -0.08439353108406067, 0.3675091862678528, -0.19775956869125366, 0.8148524761199951, 0.2244642972946167, 0.0028938017785549164, 1.0244455337524414, 1.0072154998779297, -0.22027531266212463, -0.009949653409421444, -0.5127219557762146, 0.49185827374458313, 1.496839165687561, -0.5032798647880554, -1.0339207649230957, -0.2580092251300812, -0.9405094981193542, 0.8343982696533203, -1.173079490661621, 1.317451000213623, -1.2504290342330933, 0.05225474014878273, 0.5584045648574829, -0.9423114061355591, 0.2361498475074768, 0.31877636909484863, 2.152806520462036, 0.06119948625564575, -0.9872199892997742, -0.0613078698515892, 0.8792238831520081, 0.20427046716213226, 0.6382004022598267, -0.11858067661523819, -0.9370746612548828, -0.24139128625392914, 0.9543238878250122, 0.40752726793289185, 0.2935076951980591, 1.519250512123108, -1.0402694940567017], [0.17449764907360077, -0.42058804631233215, -0.35809847712516785, -0.3214004933834076, -0.6596664190292358, -0.10947554558515549, -0.026965804398059845, 0.4715040922164917, -0.9892628788948059, 0.7105910778045654, -0.7249244451522827, -0.4421699345111847, 0.6709445714950562, 0.35376089811325073, -0.6955176591873169, -0.48133745789527893, 0.7763001322746277, 0.6197896003723145, -0.3739144802093506, -1.7178239822387695, -2.8324410915374756, 2.4387624263763428, 0.9905142784118652, 0.8507689237594604, 0.05314749479293823, -0.5949004292488098, -0.3002679646015167, -0.131034255027771, -0.05945008620619774, -1.0381557941436768, -0.4223875105381012, -0.805518627166748, 0.8578925728797913, 0.00850178487598896, 1.033847451210022, -0.6384449601173401, 0.34174636006355286, 1.150197148323059, 0.5049297213554382, -1.5785715579986572, -1.4758418798446655, 0.08039627969264984, 1.271102786064148, 1.1417336463928223, 1.2780340909957886, -2.0019822120666504, 0.010796720162034035, -0.453429251909256, -1.5993144512176514, -0.04550265148282051, -0.8781289458274841, -1.1417434215545654, 0.5927033424377441, -0.75448077917099, 0.9731408953666687, -0.12231668829917908, 1.232767105102539, 1.0469735860824585, -1.5398634672164917, 0.2604043185710907, -0.27629733085632324, 0.6863060593605042, -0.47359487414360046, 1.4002282619476318, 1.0703437328338623, -0.5703134536743164, -0.7783310413360596, 1.5716874599456787, 1.1205217838287354, -0.7026596665382385, -0.022506173700094223, -0.7640120983123779, 0.964418888092041, 0.7457543015480042, 0.9059738516807556, 0.4851309657096863, -0.6594886183738708, -1.0527175664901733, -0.030531447380781174, 0.806376576423645, 1.270656943321228, -0.9455645084381104, -0.19017843902111053, 0.1267169713973999, -1.0260913372039795, -1.552712321281433, 0.6806191205978394, -0.31563180685043335, -1.5400407314300537, -1.8378565311431885, -0.6085680723190308, 1.3949941396713257, -0.828265905380249, -0.15578024089336395, 1.2917966842651367, -0.9233802556991577, -1.2325738668441772, -0.632895827293396, 0.9644417762756348, 0.4568132162094116, -0.26967284083366394, -0.8197069764137268, -1.3367499113082886, -0.3328876495361328, -0.45405513048171997, 1.0620954036712646, -0.648285984992981, -0.7003623247146606, -0.8386072516441345, -0.624186098575592, -1.0163817405700684, 0.1796664595603943, 0.038912687450647354, 1.379460096359253, 0.9909371137619019, 0.5404417514801025, 0.08026409894227982, -0.8073279857635498, -1.246753454208374, -0.7780182957649231, 0.8665063381195068, -1.2280460596084595, -0.3940367102622986, 0.3792778551578522, 0.7125827670097351, 2.194483757019043, -0.25340205430984497, -0.5145778656005859], [0.9418514966964722, 0.31267866492271423, 1.4775745868682861, 0.43806591629981995, -1.4451225996017456, 0.23058733344078064, 2.5727615356445312, -0.44999080896377563, 1.3655840158462524, -0.8322080373764038, -1.202798843383789, -2.0608863830566406, -0.696684718132019, -1.156009316444397, 1.510896921157837, -1.5276308059692383, 0.49374741315841675, -0.0691540539264679, -1.9903388023376465, -0.47176098823547363, 1.254346251487732, 0.5577583909034729, -1.3628469705581665, 1.7240029573440552, -0.6969087719917297, -0.12831810116767883, 0.06521090120077133, -0.17547617852687836, 0.027588317170739174, -0.8026655912399292, -0.9275552034378052, 2.113532304763794, 2.6241259574890137, -1.6356018781661987, 0.504920482635498, -1.071858286857605, 0.5322747230529785, -0.8235905766487122, 0.4087521731853485, -0.8524725437164307, 1.4889616966247559, 2.119243860244751, 0.34797823429107666, -0.20302435755729675, -0.2818525433540344, 0.02539067529141903, 0.6447258591651917, 1.4048168659210205, -0.7644371390342712, 0.8252883553504944, -0.7797304391860962, -1.495802640914917, 0.9178513288497925, -0.6212079524993896, 0.8160513639450073, -0.09147563576698303, 0.7769513130187988, -0.5852640271186829, -0.10285366326570511, -1.272782325744629, 0.6481123566627502, 0.41246020793914795, -0.839303731918335, 0.3134732246398926, -1.2514857053756714, -0.8150730133056641, -0.945644736289978, -0.885356605052948, -0.5314759612083435, -0.15877079963684082, 1.558059573173523, 0.4363861382007599, 1.0487980842590332, 0.034823108464479446, -0.790683925151825, -1.0491312742233276, -1.4374934434890747, 1.433179259300232, -1.1756020784378052, -0.56447434425354, -0.6448294520378113, -0.611990213394165, 0.2871945798397064, 0.6719071269035339, 2.7470133304595947, 0.2565477192401886, 2.067957639694214, -0.00099614926148206, 0.23996871709823608, -1.847170352935791, 0.10763472318649292, -1.8622691631317139, 0.7839078307151794, 2.104952335357666, 0.4053041934967041, -0.5875129699707031, -0.5976001620292664, 3.176804304122925, -0.641869843006134, -0.5030450224876404, -0.5780019164085388, 0.5022585988044739, -1.0528888702392578, 0.23389793932437897, -1.550356388092041, -0.17585472762584686, -1.2671245336532593, 0.42289361357688904, -0.03237922862172127, -0.7981405258178711, -0.8680889010429382, 0.5155016183853149, -0.8972684144973755, 0.05130363255739212, -1.4426809549331665, 0.10217752307653427, -1.0294413566589355, 1.8834056854248047, 0.514266312122345, -0.4663086533546448, 0.5344732403755188, 1.2886828184127808, -0.05772772803902626, 0.3566615581512451, 2.3473851680755615, 0.7108229398727417, -0.6168459057807922, 0.6276038289070129], [-0.3394218981266022, 0.4909703731536865, 1.6079679727554321, -0.24097588658332825, -1.6780507564544678, -2.3213768005371094, 1.0202101469039917, 0.061085090041160583, 1.593430519104004, -1.0606328248977661, -0.45943424105644226, 2.0637600421905518, 2.190786600112915, -0.239850252866745, -1.5169143676757812, 0.6977807283401489, -0.5842373371124268, 0.8975380063056946, 0.7621697783470154, -0.21922282874584198, -1.2274972200393677, 0.5269240736961365, 0.5554114580154419, 0.40269097685813904, -1.447004795074463, -0.9705017805099487, -0.8128664493560791, 0.6770316362380981, 2.6066813468933105, -0.12749813497066498, -0.11890242993831635, 1.056465744972229, 0.3957377076148987, -0.1365700364112854, 0.8904101252555847, -0.5150516629219055, 0.374024361371994, -0.14439357817173004, -0.7559592127799988, -0.7137715816497803, -1.3631503582000732, -0.34707963466644287, 0.8043672442436218, -2.0406241416931152, -0.310844749212265, 0.014868550933897495, 1.0542197227478027, 0.4938514232635498, -0.741632878780365, -0.8042356967926025, -0.73531174659729, -0.18063752353191376, 0.6179353594779968, -0.37099677324295044, 0.39170119166374207, 0.5856470465660095, 1.0762574672698975, 1.3581215143203735, 0.4448027014732361, 0.3535093069076538, -1.2340927124023438, -0.04753313958644867, 0.17965003848075867, 0.790175199508667, 0.6422736048698425, 0.6283098459243774, 0.47892090678215027, -0.2524223029613495, -0.2945346236228943, 0.5170753002166748, -2.1421194076538086, -0.14211571216583252, -1.409616231918335, -0.04447554796934128, 1.2620521783828735, 1.002546787261963, -0.5396903157234192, 0.5960617661476135, 1.318778157234192, -1.6152799129486084, 0.5056395530700684, 1.2034400701522827, 0.9417665600776672, 0.307820588350296, 0.30058205127716064, 0.9029984474182129, 1.3970580101013184, 1.1783630847930908, 0.5464086532592773, -0.07386581599712372, -0.044683005660772324, 0.43874627351760864, -0.13529913127422333, 0.5006300806999207, -0.29931116104125977, 0.35510876774787903, 0.15806061029434204, 1.903953194618225, -0.8891941905021667, 0.8096978664398193, -0.4936264157295227, -0.12399277091026306, 0.6614392399787903, -0.3430190980434418, -2.9301724433898926, -1.247423529624939, -1.375345230102539, -1.0961321592330933, -0.11409423500299454, -0.01753068156540394, -2.4854066371917725, 0.038561660796403885, -0.4927554428577423, 0.4726150929927826, -0.9916999936103821, -1.137372612953186, -0.1473761945962906, -1.661878228187561, 1.2213811874389648, 0.829614520072937, 2.007671594619751, -0.5067381858825684, -0.6347163319587708, 0.708502471446991, -1.8411321640014648, -0.35604792833328247, 1.0958266258239746, -0.8185522556304932], [-1.9623222351074219, -0.8161748051643372, 0.38723617792129517, -2.2224373817443848, -0.9525051116943359, -0.46670934557914734, -0.39721038937568665, -0.5807802677154541, 1.1206448078155518, 0.2029733508825302, 0.02875634841620922, -1.696415901184082, -0.28417348861694336, -0.6581710577011108, 2.125197649002075, 0.698691725730896, -0.3556308448314667, 1.0167427062988281, -1.7459036111831665, 1.1036924123764038, -1.5069308280944824, 0.9449304342269897, -1.8933707475662231, -1.2452528476715088, 1.057861089706421, -1.1986721754074097, -1.7062175273895264, -1.852461338043213, -1.5401983261108398, 0.751668393611908, -0.9803422689437866, -1.008901834487915, 2.314018726348877, -0.03552466258406639, -0.12752491235733032, 0.7014623284339905, 0.8336613774299622, -0.4075104594230652, 0.5055638551712036, 1.4998266696929932, -0.2909395396709442, -0.24787218868732452, -0.7563929557800293, 0.23341260850429535, -2.095113515853882, -0.4684109687805176, -1.94840407371521, 0.6540287733078003, -1.1884593963623047, 0.640013575553894, 1.3973392248153687, -0.8660524487495422, -0.6628223061561584, -0.2970806062221527, 0.23156972229480743, -1.4343513250350952, -0.34380778670310974, -0.8834982514381409, 1.3709856271743774, 0.6208601593971252, -0.39678919315338135, -1.2673925161361694, 1.971327304840088, 1.2419453859329224, -0.3282080292701721, 1.755052089691162, 0.16061195731163025, -2.0835776329040527, 0.8297240138053894, 0.04458416998386383, 0.7864863872528076, 0.13529200851917267, -0.24075713753700256, -1.8803170919418335, 1.0241963863372803, -0.5825755596160889, 1.1658525466918945, -1.6881282329559326, 0.01892426609992981, -0.9242952466011047, 1.1720083951950073, 0.9668518304824829, 0.260159432888031, 2.6610989570617676, 0.6777785420417786, 2.3781261444091797, 0.1249580904841423, 0.3990309238433838, 0.18999403715133667, -1.0424766540527344, -0.7544176578521729, 0.16357293725013733, -0.07452049851417542, -0.7197777032852173, 0.3391772210597992, -0.07839503884315491, 0.6643289923667908, -0.3431321084499359, -1.2412315607070923, -1.513963222503662, -1.273129940032959, -0.3973549008369446, 0.8813380599021912, -0.6406447291374207, 0.710952639579773, 0.25259625911712646, -1.6562327146530151, -0.17893806099891663, -1.0137600898742676, 1.3092643022537231, -0.1291685253381729, -0.3743002712726593, -0.14036132395267487, -0.2038399577140808, 0.799121081829071, 0.07066088169813156, 0.07233522087335587, 0.6227564811706543, 1.1589269638061523, 0.289545476436615, -1.4287693500518799, -0.2849096953868866, 0.5000072717666626, -0.18254928290843964, -1.0247976779937744, -1.353000283241272, 0.7316323518753052, -1.9208157062530518], [-0.24187994003295898, -1.1663753986358643, 0.6576030850410461, -1.2568519115447998, 2.0251548290252686, 0.06681890785694122, -0.2585166394710541, -0.3822377026081085, -0.11483972519636154, 0.8395837545394897, -0.6030113101005554, -1.7091901302337646, -1.9011082649230957, 0.841423749923706, 1.2449469566345215, 1.0709376335144043, -1.1741092205047607, -0.5496765375137329, -2.0075511932373047, 0.8492937088012695, 0.3988974392414093, 0.3654673397541046, 1.642512559890747, 0.22657857835292816, 1.427675724029541, 0.21240568161010742, -0.9041662216186523, -0.4786393642425537, 0.9688528180122375, -0.6207171678543091, -1.611840009689331, -0.29603153467178345, -0.30806005001068115, -0.8962386250495911, 0.2455914169549942, -2.001037359237671, 0.3268316388130188, 0.7499051690101624, 0.17003248631954193, -4.045924663543701, -1.8862203359603882, 0.4234234094619751, -1.5594977140426636, 1.588800072669983, 0.8925992250442505, -2.0458950996398926, -1.769933819770813, -1.4568783044815063, -0.06766156852245331, 0.8309690356254578, 0.008522490039467812, 0.2349996566772461, -1.139669418334961, 0.49178406596183777, -0.40085917711257935, 2.3470871448516846, -0.47030410170555115, -0.43724700808525085, -0.46984580159187317, -1.376220464706421, -0.2915321886539459, -1.4355860948562622, 1.1951611042022705, -1.3218227624893188, -0.019933102652430534, 1.893602728843689, 0.7491469979286194, 0.18424968421459198, 0.41185420751571655, 2.0916450023651123, 2.8901596069335938, -0.922329843044281, -0.3775773346424103, -0.11859997361898422, -0.3513840436935425, 0.2628585398197174, 0.7806233763694763, -0.7430657148361206, 0.9575293660163879, -0.8594672679901123, 0.18387213349342346, -0.5302577614784241, 0.09870412200689316, -0.30591875314712524, 1.0850260257720947, 0.30627691745758057, -1.3542193174362183, 0.23684147000312805, -0.9379134774208069, -0.9121152758598328, -1.6874136924743652, -1.381717324256897, -0.6815829873085022, -0.25506001710891724, -0.24350641667842865, -0.1303885281085968, -0.7838118672370911, 0.13609927892684937, 0.5457555055618286, 0.47447332739830017, -0.7640816569328308, -0.30884602665901184, 0.4142025411128998, -0.6591907143592834, 0.6422398090362549, 1.4991106986999512, 0.8901304602622986, 0.9325999021530151, -1.6861138343811035, -0.6164001822471619, 0.30775707960128784, 1.3839627504348755, 1.5166490077972412, -0.7040191292762756, 1.1024936437606812, -1.112505316734314, -0.18756763637065887, -0.7587805986404419, 0.8265836834907532, -2.790602684020996, -0.28490498661994934, 0.3330969512462616, 0.3837645351886749, 0.7116357684135437, -0.3583633005619049, -0.5127884745597839, -0.031973160803318024, 0.7448720932006836], [0.6532180905342102, 1.6827763319015503, -1.266175627708435, -1.505984902381897, -1.1777998208999634, 0.12010639905929565, -0.0838530957698822, 0.8523781895637512, -0.02967192232608795, 0.3728733956813812, -1.610158085823059, 0.2569698393344879, 0.9330841898918152, -1.124373197555542, -0.332680344581604, 0.27229171991348267, -1.9756072759628296, 0.9710624814033508, 0.501808226108551, 0.7680586576461792, -1.177130103111267, 0.26052936911582947, 2.1164159774780273, -0.5147847533226013, -0.6581066250801086, -0.7398517727851868, 0.5188366174697876, 1.722898244857788, 0.888451337814331, -1.4304132461547852, -0.40482455492019653, -0.878840982913971, -0.8811495900154114, 1.8879703283309937, 1.0516544580459595, -0.641635537147522, -0.50025874376297, 0.06327640265226364, 1.076896071434021, 0.11231015622615814, -0.3442024290561676, 0.5310136675834656, 1.1284509897232056, 0.9859080910682678, -0.00921020656824112, -0.06725793331861496, -0.6182279586791992, 0.2932484447956085, 1.0170902013778687, -0.5701255798339844, -0.30915260314941406, -1.453860878944397, 0.4641566872596741, 0.6859620809555054, 0.3502674698829651, -2.5950427055358887, -1.5481107234954834, -1.1189192533493042, -1.0580233335494995, -1.3218393325805664, 0.37936931848526, -1.9763890504837036, 1.7921135425567627, 0.6527624726295471, 0.47161880135536194, 0.888231635093689, 0.4934755563735962, 0.9623773097991943, 0.6956443190574646, -0.9518706798553467, -1.5963865518569946, -0.6748039126396179, -0.09293225407600403, 0.09800926595926285, -1.5255143642425537, -0.599753737449646, -1.9363093376159668, -0.122705839574337, 0.04240158200263977, -0.2531425356864929, 0.9591771960258484, -1.675723671913147, -0.5070800185203552, 1.7639611959457397, 0.1447097212076187, 0.27633315324783325, -0.4993784725666046, -1.6959919929504395, -2.081573009490967, -0.4827296733856201, 1.890043020248413, -1.4715572595596313, -1.6140495538711548, -0.3503938913345337, -1.1637239456176758, 0.16039814054965973, 1.7323899269104004, -1.3641666173934937, -0.41762349009513855, -0.16709449887275696, 1.9042551517486572, -0.6002120971679688, 0.628052830696106, 0.09845861047506332, -0.12622739374637604, 0.6236454248428345, 0.15487238764762878, -0.7321561574935913, -0.1512829065322876, -0.49943122267723083, -0.14724759757518768, 0.4615468382835388, -0.5741880536079407, -0.11645589023828506, -0.3494662344455719, -0.785100519657135, 1.7502580881118774, -0.2618153691291809, 1.6812716722488403, 0.7967262864112854, 0.4361429214477539, -1.2985771894454956, 1.8582459688186646, -0.481916218996048, -0.28509148955345154, 0.5985035300254822, -0.5622817277908325, 1.1435030698776245]], [[1.3588407039642334, -0.8940200209617615, 0.7032580971717834, -0.4060584604740143, -0.7387025356292725, -0.09200561791658401, -1.1053495407104492, -0.10646368563175201, -0.23977868258953094, -0.5783130526542664, 0.8677136898040771, 1.0694655179977417, -0.37183451652526855, -1.3722583055496216, 1.4590933322906494, -1.4110863208770752, 0.8198585510253906, -0.25479596853256226, -0.16503028571605682, -0.0890587717294693, 1.0898165702819824, -0.8994613289833069, -0.06565464287996292, 0.4788726568222046, 0.5831054449081421, -0.0013329458888620138, 0.08345501869916916, -0.37871918082237244, 0.9645920395851135, 0.5682857036590576, 0.8141791224479675, -1.5711746215820312, -0.09075841307640076, 0.031396567821502686, 0.812704861164093, 0.5814991593360901, 0.32762032747268677, -0.33341601490974426, 1.3398585319519043, -1.1444499492645264, 0.09374206513166428, 2.1914076805114746, 0.07085126638412476, -0.2774580419063568, -0.1580071747303009, 0.5716046690940857, -0.7996092438697815, 0.3751969635486603, 2.085592031478882, -0.9269410967826843, 0.7804933190345764, 0.3705233633518219, 0.27359461784362793, -1.5829339027404785, -0.6909723281860352, 1.6748579740524292, 0.07238321006298065, 1.7436116933822632, -0.20450706779956818, 0.6823082566261292, -0.08802427351474762, 0.8731102347373962, -0.08690885454416275, 1.0684716701507568, 0.2945745885372162, 1.5275373458862305, 0.7720266580581665, 0.03754744678735733, -0.015109177678823471, -0.1706475019454956, -0.2577776610851288, -0.6071265339851379, 0.10454276949167252, -0.10154769569635391, 0.9888906478881836, 0.950516939163208, 0.1773560643196106, -0.7878867387771606, 0.5680108666419983, 0.0961962342262268, -0.6823310256004333, -1.8347982168197632, 3.50961971282959, 0.06661128997802734, 0.4492209553718567, -0.3927927017211914, 0.39797720313072205, -0.15555623173713684, -1.4790019989013672, 2.326899290084839, -0.11702420562505722, -0.3773705065250397, -0.7252646684646606, 0.2775074541568756, 0.4076237082481384, 1.1495678424835205, -0.0044878944754600525, 0.7161721587181091, 0.9838883876800537, 0.3242036998271942, -1.871787667274475, -0.4646340608596802, -0.5957186818122864, -0.8328919410705566, -0.08630518615245819, 0.29536086320877075, 0.6780152320861816, 0.6451410055160522, -0.10469714552164078, -0.8383190631866455, 0.3269316256046295, 0.7539685964584351, -0.851345956325531, -1.0417903661727905, 0.20001453161239624, -1.7135969400405884, 1.8971598148345947, 0.6498397588729858, -0.6018008589744568, 0.5485985279083252, -1.2478536367416382, 0.49489226937294006, 0.13734519481658936, 0.5276466012001038, -0.4342477321624756, 0.2222379744052887, -0.8615604639053345, 1.4082186222076416], [-0.9570861458778381, 0.0511261448264122, -2.3458430767059326, 0.21859949827194214, 1.4205764532089233, -1.3929678201675415, 1.1857328414916992, -0.37366020679473877, 1.9342707395553589, 1.2304530143737793, 0.5542507171630859, 0.7156486511230469, 1.1123015880584717, 1.2777254581451416, 0.4364641308784485, 0.779918909072876, -1.6277889013290405, 0.5279845595359802, -0.7310157418251038, -1.2854828834533691, 1.029030203819275, -1.4870339632034302, -0.3029201626777649, -0.16876521706581116, 0.28874725103378296, 0.8843345642089844, 0.2876701056957245, -0.18590669333934784, 1.9189034700393677, 0.358132004737854, 0.41053420305252075, 2.0577311515808105, 1.3777830600738525, -1.4041948318481445, -0.4099092483520508, 0.35181549191474915, -0.8890153169631958, 0.28188326954841614, -2.1116538047790527, 0.6196509599685669, 1.4412932395935059, -1.563517451286316, 0.3660646975040436, -0.09011312574148178, -1.3007631301879883, 0.42677077651023865, -1.707776665687561, 1.0080411434173584, 0.09003225713968277, -0.3613545298576355, -1.378495216369629, 0.794044554233551, 0.6934241652488708, 0.8590596914291382, -0.15857723355293274, -0.3065517842769623, -0.6707712411880493, 1.4311542510986328, -0.1018529012799263, -1.2419183254241943, 0.5117164254188538, -1.1241787672042847, 0.11239601671695709, -0.5286856889724731, 0.18363365530967712, 0.26329225301742554, 0.29750218987464905, 0.04740985110402107, -0.06532439589500427, 1.309916377067566, -0.3644947111606598, -0.7358611822128296, -0.815807044506073, 2.869295358657837, 0.9204905033111572, -0.0036011661868542433, 0.2675279378890991, -2.491382598876953, -0.4612393081188202, -1.1986806392669678, 1.1704938411712646, -0.949627697467804, 0.16596630215644836, -1.3355087041854858, 2.08909010887146, 0.11973174661397934, -0.35035520792007446, -0.6890090107917786, 0.26294371485710144, -1.0610686540603638, -0.1683613657951355, -1.6137197017669678, -1.4073998928070068, 2.348320722579956, -1.613856554031372, 0.5934296250343323, -0.9392083287239075, -1.8147205114364624, -2.3239564895629883, -0.5353448987007141, 2.2120397090911865, 1.4554080963134766, 0.6157513856887817, 1.0098631381988525, -0.6525734066963196, -2.0390729904174805, 0.5959789752960205, 1.4300285577774048, -0.40545278787612915, 2.1610186100006104, 1.0653215646743774, -1.0289826393127441, -1.2801473140716553, -1.2266545295715332, -0.5877463817596436, 0.6625896096229553, -0.9019464254379272, -0.09182002395391464, 0.08721987158060074, 0.6164128184318542, -0.6548107266426086, 0.9949448704719543, 1.2692863941192627, -0.26090192794799805, -0.5515986680984497, -0.9379642605781555, 0.3421038091182709, 1.2758203744888306], [-0.17261868715286255, 0.20043915510177612, -0.3679063022136688, -0.9346374273300171, -1.7733193635940552, 0.8156822919845581, 1.2165781259536743, -0.16506990790367126, -0.22039198875427246, -0.8899297714233398, 0.560794472694397, 0.8934941291809082, -1.7630202770233154, -1.0286003351211548, 0.751985490322113, 0.2609427869319916, -0.7050740122795105, -1.1509392261505127, 0.3127518892288208, 0.9427196383476257, 0.552926778793335, 2.0932204723358154, 0.5466293692588806, 0.02281549572944641, -0.8862991333007812, -2.3437423706054688, -0.9061444401741028, -1.2757856845855713, -0.750063419342041, 0.8469520807266235, 0.388727068901062, 0.9772225618362427, -0.3452570140361786, 1.4560377597808838, 0.4413166344165802, 1.1170868873596191, -0.3674062192440033, -0.6830200552940369, 0.23710840940475464, -0.44692176580429077, -0.527081310749054, 2.069133758544922, -0.8156982660293579, 0.41770246624946594, -0.4477415084838867, -0.5132603645324707, -0.19739432632923126, 0.6912837028503418, -1.0818567276000977, 0.26850900053977966, 0.5200964212417603, 0.04056257754564285, 0.48325926065444946, 0.4937412440776825, -0.7971371412277222, -1.0293731689453125, 0.9790145754814148, -0.6825783848762512, -1.064732551574707, 0.4159749150276184, -0.34315112233161926, -0.16065435111522675, 0.4311656057834625, 1.5385438203811646, -1.246506690979004, -0.22625064849853516, -0.5901479721069336, 0.24806083738803864, 0.19300825893878937, 0.6358871459960938, 1.4257197380065918, 0.3146328926086426, -0.44865667819976807, -1.467775583267212, 0.48842379450798035, -1.0524455308914185, 1.1746467351913452, -0.5099861025810242, 0.7062817811965942, 0.9111464023590088, -1.7030689716339111, -0.9451558589935303, 0.12397190928459167, 1.3392313718795776, 0.37127336859703064, 0.12013638764619827, 0.046194687485694885, -0.722991943359375, -1.199596881866455, -1.3815953731536865, -0.056756943464279175, 0.2814778983592987, 0.2832616865634918, 0.16754555702209473, 0.32801637053489685, 1.2848602533340454, -0.151789590716362, 0.8431714177131653, 0.6917118430137634, -0.15100526809692383, -0.35625943541526794, 0.06628688424825668, -0.1192634180188179, -0.7392281889915466, -0.6841719746589661, -1.446999192237854, 0.14121192693710327, 1.5897924900054932, 0.8144675493240356, -0.5283125042915344, 1.129733920097351, 0.7383137345314026, -1.5385066270828247, 0.5041980147361755, -2.212944269180298, -0.10017142444849014, 0.09350971132516861, -2.6339051723480225, -0.018012847751379013, -0.2304278016090393, 0.29615893959999084, -1.2577067613601685, -0.8707019090652466, -1.4287629127502441, -1.2140569686889648, -0.19114285707473755, 0.25207239389419556, 0.40942251682281494], [0.8261477947235107, 0.9853004217147827, 0.1062726378440857, -0.1731179654598236, 0.5508933067321777, -1.4066393375396729, 0.35766229033470154, 1.3713066577911377, -0.4542192220687866, -0.5630736947059631, -0.10369396954774857, -0.8330469131469727, -1.9450006484985352, 0.04654623568058014, 0.4052262306213379, -0.2959379255771637, 0.32232674956321716, -0.46372440457344055, 0.781370997428894, -0.8972991108894348, 0.691856861114502, 0.5920681953430176, -0.8734692931175232, -0.07340139150619507, 0.5609229207038879, 0.34384623169898987, -0.15543495118618011, -1.5348386764526367, 0.8719362020492554, 1.888881802558899, 1.388547658920288, -1.2041994333267212, -2.008997678756714, -0.49580731987953186, 1.114058494567871, 0.12283523380756378, 1.028516173362732, 0.8846874237060547, 0.7984520196914673, -1.2160353660583496, 0.24376048147678375, -2.535959005355835, -0.3469434380531311, 1.0414584875106812, -0.9981998205184937, -0.25812986493110657, -0.6239657998085022, 0.8122527599334717, 0.4226095974445343, 0.3098393976688385, -0.23513339459896088, 2.277188539505005, -0.05076586455106735, -0.007603050209581852, -1.2495548725128174, -1.65414297580719, 0.6995977163314819, -0.7040262818336487, 1.1070882081985474, -1.1097995042800903, 1.1781724691390991, 0.9822983741760254, -0.9160623550415039, 0.5734253525733948, 0.36948466300964355, -1.9149317741394043, 2.1118063926696777, 0.6420106887817383, 0.8626075387001038, 0.17464037239551544, -1.1467854976654053, 0.05795904994010925, -0.0068886904045939445, -1.3845210075378418, -0.4126032292842865, -0.21327370405197144, -0.5323675870895386, -0.9402385354042053, -0.3390536606311798, 1.0244662761688232, -0.11541064083576202, 1.6763296127319336, -1.6973260641098022, -0.704491376876831, 1.5117342472076416, -1.761282205581665, 0.8942644000053406, 1.0993809700012207, 1.3569550514221191, -0.03240090608596802, -0.5271955728530884, 0.3237333297729492, -0.20231883227825165, 1.3141472339630127, 0.777426540851593, 0.06518781185150146, 1.0681941509246826, 1.9572453498840332, -0.7019614577293396, -0.3389461040496826, 0.9322453737258911, 0.8578674793243408, 1.0508441925048828, -0.6539286971092224, -1.6130980253219604, 1.8687859773635864, 0.4498198330402374, 0.6468101739883423, -0.48389214277267456, 1.32075035572052, 0.12272153049707413, -1.284658432006836, -0.8891342878341675, 1.8067588806152344, 0.8129478096961975, -1.6183338165283203, 0.14490754902362823, 0.3754226267337799, -0.48366233706474304, 1.586320400238037, -0.8776975274085999, 0.09755392372608185, 0.19916903972625732, 1.1772888898849487, 1.2393711805343628, -0.35538867115974426, -1.269670009613037, 1.335431456565857], [-0.003598729381337762, 1.036058783531189, -0.1750335991382599, -1.3173717260360718, -0.20763254165649414, -1.192057490348816, 1.677846074104309, -0.7771520614624023, 0.20908881723880768, -0.49822574853897095, -0.15219265222549438, 1.1501367092132568, -1.4362775087356567, -0.7394533753395081, 1.3513826131820679, 0.5041249990463257, -0.4895818829536438, 0.7455292344093323, -0.44764819741249084, 0.6878475546836853, -0.054266009479761124, 0.6470101475715637, -1.6917128562927246, 1.1649237871170044, -0.9821507334709167, -0.6547773480415344, -0.22297699749469757, 0.1922278255224228, 0.05961838364601135, 0.31791993975639343, -0.338821142911911, 1.4407175779342651, 0.4393041133880615, -1.52442467212677, -0.29170194268226624, -0.153982475399971, -0.2711471915245056, 1.4992363452911377, -1.6759068965911865, 0.9359501004219055, 1.2649246454238892, -0.9207797646522522, 0.4177493453025818, 0.39619001746177673, 1.1767570972442627, 0.4779977798461914, 0.5968721508979797, 1.1379222869873047, -0.9076275825500488, -0.9788743853569031, 0.5072726011276245, -1.3897452354431152, 0.5695138573646545, -1.6109040975570679, -0.9898710250854492, 0.16227911412715912, -0.8772466778755188, 0.8102885484695435, -1.8337152004241943, -1.1542572975158691, 2.1980111598968506, -0.31103867292404175, -0.12372051924467087, 0.0879574790596962, -2.3815572261810303, 0.2514191269874573, -1.5970450639724731, -2.4825520515441895, -0.38663601875305176, 0.5049338936805725, 2.1902153491973877, 0.9401146769523621, 1.0229769945144653, 1.7136098146438599, 0.20188049972057343, -0.4404893219470978, 0.9728792309761047, 0.24924027919769287, 0.3394363820552826, 0.34078550338745117, 0.9859228134155273, 0.09216863662004471, 0.3243654668331146, -0.5276356935501099, -0.8865974545478821, -1.1812207698822021, -0.3368299603462219, -0.9900283217430115, 0.3455866575241089, 2.232011079788208, -0.08750905841588974, -0.45181915163993835, -0.47232118248939514, -0.046078525483608246, 0.7844053506851196, -0.0701790302991867, 0.7773722410202026, -1.7129032611846924, -0.5167950987815857, -2.6896860599517822, -0.6327612996101379, -0.14266827702522278, -0.991930365562439, -0.23438268899917603, 0.8935674428939819, -2.3702871799468994, -0.6537204384803772, -1.520796775817871, 0.9482905864715576, -0.8344704508781433, -0.17404557764530182, 1.4454599618911743, -0.9295166730880737, -2.379185438156128, -0.3645985424518585, -0.27694109082221985, -0.685673177242279, 2.041943073272705, -0.6835929155349731, -0.3207930326461792, 0.5893713235855103, 0.10323252528905869, -1.018722653388977, -1.4920368194580078, -0.2561434507369995, -0.020801886916160583, 0.15557824075222015, -0.4410657286643982], [-0.32721731066703796, -1.5617449283599854, -0.31632983684539795, 0.5397385954856873, 1.5355335474014282, 1.2277077436447144, -0.9840682744979858, 0.0531298965215683, 0.6622178554534912, -0.41903725266456604, 1.134158730506897, -0.10144665092229843, 1.0207067728042603, -0.5743851661682129, -1.4964289665222168, 0.28177377581596375, -0.7775097489356995, 0.6044058799743652, -0.23328667879104614, -1.239615559577942, 0.3301982283592224, -1.4648326635360718, 1.691723108291626, 1.6539793014526367, 0.39160799980163574, 0.07672198116779327, -1.3875279426574707, -0.6347293257713318, -0.842465877532959, 0.18785962462425232, 0.4469718337059021, 0.9329894185066223, -1.7223279476165771, 0.46848294138908386, -0.6798224449157715, 0.07482943683862686, 0.48745325207710266, 0.7411504983901978, -1.882357120513916, -1.2513377666473389, 0.8775248527526855, 1.9325547218322754, -0.06997323036193848, -0.5597315430641174, -0.0826805979013443, 0.8854740262031555, 0.1386559009552002, -1.103879451751709, 0.12081602215766907, 0.21373578906059265, 0.36729681491851807, 0.04491487890481949, 0.4125925898551941, -0.08706915378570557, -1.3797674179077148, -0.05379614979028702, 0.3443841338157654, 3.183534622192383, -0.512876033782959, -0.8275656700134277, 1.072690725326538, -0.7636301517486572, 2.6725921630859375, -0.7697286605834961, -0.7246981263160706, 1.0757355690002441, 0.8494137525558472, -0.2311442792415619, -0.43956834077835083, -0.757767379283905, -0.8569198846817017, -0.4657377302646637, 0.2578531801700592, -0.558523952960968, 1.869315505027771, -0.4930146634578705, 0.5089637637138367, -1.0846999883651733, 0.4130387306213379, -0.1300082951784134, -0.022958777844905853, 1.0311919450759888, 2.435513496398926, 1.3310610055923462, -1.2651004791259766, -1.3723007440567017, -0.04020223766565323, 0.5206061601638794, -0.6938790082931519, 1.310788631439209, -2.1009552478790283, 0.9585485458374023, 0.7104885578155518, 0.015613153576850891, -1.8686553239822388, -1.1079869270324707, 1.9429508447647095, 0.8408849239349365, -0.09050611406564713, -0.5653496384620667, 1.2040035724639893, -0.1981031745672226, 1.1683558225631714, -0.6823285818099976, 0.45320525765419006, 0.5539640784263611, 0.16227875649929047, -0.4506758451461792, 0.7005773186683655, 0.3792673945426941, 0.31128183007240295, 2.6035468578338623, 0.2549251317977905, -1.7620036602020264, -0.9700939655303955, -1.6624653339385986, 0.30724388360977173, -0.39957094192504883, -0.5796273946762085, -0.16168951988220215, -0.8055117726325989, 0.7656619548797607, 0.6081414818763733, 0.6414302587509155, 1.3655813932418823, -0.47428447008132935, -0.10007737576961517, 0.7435048222541809], [0.9298722743988037, 1.6742931604385376, -0.5592643618583679, -1.6910037994384766, 1.8489806652069092, -1.094262719154358, 1.7995177507400513, -0.618152916431427, 0.08893107622861862, -0.7550157308578491, 0.47077852487564087, -1.2605279684066772, -0.22543509304523468, 0.7881015539169312, -0.2639569044113159, 0.031955912709236145, -1.9331998825073242, 0.7696682810783386, 0.5999181270599365, 0.5327279567718506, 0.7583993673324585, -0.15571628510951996, -1.2453006505966187, 1.0789910554885864, -1.4109368324279785, 0.7247337102890015, 1.8213430643081665, 0.949690043926239, 0.6973840594291687, 0.7790740132331848, -0.1592170149087906, -1.0576834678649902, 0.23306970298290253, -0.5423738956451416, -2.1968672275543213, 0.7051211595535278, 1.8137742280960083, -0.4340756833553314, -0.6590945720672607, 1.0741252899169922, -0.2458391636610031, -1.4544092416763306, -1.3330094814300537, 0.8117086887359619, -1.1613234281539917, 0.07737599313259125, 1.0269259214401245, 0.3027192950248718, 0.2341337651014328, 2.8259470462799072, 0.26311707496643066, 0.3794570565223694, 0.8537595272064209, -0.9955430030822754, -0.3687642812728882, -0.34023821353912354, -1.4512922763824463, 0.24541877210140228, -0.6356022357940674, -0.8573484420776367, -1.3627429008483887, -0.17379534244537354, -1.0459320545196533, 0.5316285490989685, 0.3389435410499573, 2.269517421722412, 0.08237893134355545, 0.36498692631721497, 0.914434552192688, -0.18697218596935272, -1.118149757385254, -0.13449415564537048, -1.094791293144226, 0.7233180403709412, 0.6357658505439758, 0.3927607834339142, 0.6672818064689636, -0.39983925223350525, -1.2693090438842773, 1.0744340419769287, 0.0640215203166008, -1.7039368152618408, -1.5569430589675903, -0.7126011848449707, 0.49168241024017334, 0.965319812297821, -1.213629126548767, -0.7141618728637695, 0.4280342757701874, 0.030177004635334015, -0.8808738589286804, 1.1551121473312378, -0.10669058561325073, 0.2842661440372467, -0.7124024629592896, 0.5532975792884827, -1.6882396936416626, 0.8915156126022339, -0.3746233284473419, 0.6186384558677673, -0.04977613314986229, 0.46104753017425537, 2.0704121589660645, -1.0763295888900757, 0.32269272208213806, -0.11508551239967346, 0.5592646598815918, 0.06934776157140732, 0.5281813740730286, -1.4506176710128784, -0.8533346056938171, 0.6949751377105713, 0.9717360138893127, -1.259939432144165, 0.3351733684539795, -1.2656877040863037, 1.3402657508850098, -1.9694044589996338, 0.9470154643058777, 1.1687427759170532, 0.15736062824726105, -1.52658212184906, -0.9693267345428467, -1.1498701572418213, -0.8809652924537659, -1.1259241104125977, 0.7592389583587646, -0.4487086236476898], [0.9399380087852478, -0.36007288098335266, -0.6977975964546204, 1.8690464496612549, -1.1659008264541626, 0.7030801177024841, 2.192063808441162, 1.500997543334961, 0.519305408000946, 2.0909483432769775, 1.273512840270996, -1.9359890222549438, -1.0500491857528687, -0.902418851852417, -1.6662805080413818, 0.4418565332889557, -0.22646793723106384, 0.3678871691226959, -0.5549879670143127, 0.7662323117256165, -2.1142477989196777, 0.46352526545524597, 0.7955989241600037, -0.014310934580862522, -0.17422185838222504, 0.8451220989227295, 1.425441861152649, 1.1596124172210693, 1.0722471475601196, 0.9312906265258789, 1.0468090772628784, -0.45131492614746094, 1.6386799812316895, -1.3528053760528564, -1.3277772665023804, -0.009817736223340034, 0.12109430134296417, 0.010428530164062977, -1.2656221389770508, -2.2104291915893555, -0.041302673518657684, 0.1447490006685257, 0.007336149457842112, -0.703832745552063, -0.35692745447158813, -0.6757875680923462, 0.6725867390632629, -0.9867855906486511, 1.5095303058624268, 0.0108192702755332, -0.6401270627975464, -0.2233353555202484, -1.4309853315353394, -0.02194385789334774, 0.80947345495224, -0.3244199752807617, 1.4822384119033813, 0.2625811994075775, -0.2884472906589508, 0.30633971095085144, -0.3449561595916748, 0.9066895842552185, 1.4137816429138184, 2.00246524810791, 1.3243647813796997, -0.9857863783836365, 0.21256299316883087, 0.3876025378704071, 0.2857241928577423, -0.30394190549850464, -0.8573317527770996, -1.231953740119934, 0.38969728350639343, 0.23670591413974762, -1.3812505006790161, 0.29867541790008545, 0.3924146592617035, -0.09236571192741394, 1.6152687072753906, 0.28748735785484314, -0.42888376116752625, 2.0206246376037598, -0.25472205877304077, 1.07966148853302, 0.059014398604631424, 0.49308252334594727, -0.13196635246276855, 0.21525928378105164, 0.8796159625053406, -0.7017198801040649, 0.22668592631816864, 0.8711309432983398, 1.2322430610656738, 0.920640230178833, -0.33072948455810547, 1.6819456815719604, 2.73384952545166, 0.24650289118289948, 1.4745988845825195, 1.0641670227050781, -0.8873818516731262, 0.05814467743039131, -0.799378514289856, 0.9614651799201965, 0.34682831168174744, 1.1660414934158325, -1.039594054222107, -0.11591167002916336, -0.06423300504684448, 0.9574042558670044, -0.9612298607826233, -0.015741974115371704, 0.2833106815814972, -0.4944816529750824, 0.6132242679595947, 0.22624696791172028, -0.46729016304016113, -1.2868045568466187, 0.7748270034790039, -1.2761338949203491, 1.406599760055542, -0.9385808110237122, 1.6983736753463745, -0.35370370745658875, 1.278886079788208, -0.5480863451957703, 1.1900990009307861, 0.8168410658836365], [-1.0499582290649414, 0.2427392601966858, -0.6595832705497742, -0.4033328890800476, 2.29170560836792, 0.16513770818710327, -0.06594618409872055, 0.6671019792556763, -1.3572654724121094, -0.1068035364151001, -0.008255441673099995, 0.8376089334487915, 0.6456258893013, 0.3967837989330292, 0.4258638024330139, 0.7932211756706238, 0.0957530066370964, -0.3886510729789734, 0.09607475996017456, -0.033550504595041275, 0.7912411093711853, 0.4956003725528717, -0.024434911087155342, 1.9045953750610352, -0.07082835584878922, -0.5923429131507874, 1.7790955305099487, 0.47306933999061584, -0.9531071782112122, -0.5077885985374451, -0.1845889836549759, -2.4932198524475098, -0.9210841059684753, 0.3312142789363861, 2.364763021469116, 0.021439291536808014, -0.5492479205131531, -1.3073227405548096, 1.2888497114181519, -0.7587440609931946, -1.3803375959396362, 2.7350926399230957, -0.5334253907203674, 1.8601875305175781, -0.825465202331543, 1.2599655389785767, -0.8000994324684143, 0.3772536516189575, 0.41698455810546875, 2.350209951400757, -1.2798023223876953, 1.4944911003112793, 0.4053647518157959, -0.2503795623779297, -0.4357587695121765, 0.4497854709625244, -1.0541857481002808, -1.5725263357162476, -0.2904077172279358, 0.48754170536994934, -1.0904816389083862, 0.580985963344574, 0.8337967991828918, -1.7710152864456177, -0.07896429300308228, 1.1825395822525024, -1.8439342975616455, 0.1343563199043274, 0.2907283902168274, 0.8510025143623352, 0.5245332717895508, -0.41160306334495544, -1.0110503435134888, 1.7754004001617432, -1.8488469123840332, -0.6360413432121277, 0.893592119216919, 1.568735122680664, -0.25805315375328064, 0.7853512763977051, 0.6909875869750977, 0.5695207715034485, -1.4056429862976074, 1.007069706916809, -1.5367863178253174, 0.5982729196548462, 0.7594559192657471, -0.4795812964439392, 0.517802894115448, -0.5199058055877686, 0.22526094317436218, 0.6516349911689758, -0.8299660086631775, 0.8479270935058594, 0.208889901638031, -1.6323390007019043, 1.4504191875457764, -1.1830024719238281, -1.4210021495819092, -0.56654292345047, -1.1631944179534912, 0.384823739528656, -1.386240005493164, -0.31262099742889404, 0.5082283616065979, -1.557624101638794, -1.3919628858566284, -2.388975143432617, 0.08363284915685654, 0.30602389574050903, 1.3136683702468872, -0.5681697726249695, 1.0577107667922974, -0.6442722082138062, 1.0666990280151367, 1.47490656375885, -0.9204370975494385, 0.7011523246765137, -0.4801540672779083, 0.5793871879577637, -1.692407488822937, 0.8110135793685913, -0.19449065625667572, -0.6344151496887207, 1.530342936515808, 2.063724994659424, 1.8216005563735962, 0.25959542393684387], [-0.7141460180282593, -1.0977671146392822, 0.22736063599586487, -0.4501512050628662, -0.2626335322856903, -1.6222659349441528, -0.9087308049201965, -0.8128820061683655, -0.8807474970817566, -1.1794061660766602, 0.377475768327713, 1.1295801401138306, -0.09292585402727127, -0.1554197072982788, 1.077959656715393, 1.290254831314087, 0.8050537705421448, -0.010089659132063389, -1.2982155084609985, 0.24123182892799377, -0.9224644899368286, -0.07097122073173523, -0.7510167360305786, -0.6846615076065063, -0.7211248278617859, 0.7836668491363525, 0.6008221507072449, 0.6794331669807434, 0.21881136298179626, 0.02707221545279026, -0.21515429019927979, -0.023565899580717087, 0.5179435610771179, -0.05786178633570671, 0.30959171056747437, 1.6662218570709229, -2.397793769836426, 1.7969260215759277, -1.2738627195358276, -0.31233441829681396, 0.6552245616912842, -0.35343170166015625, -0.045388948172330856, -1.5364965200424194, -0.558178186416626, -0.07623095065355301, -0.16692791879177094, -1.1725441217422485, -0.282215416431427, -0.1153411865234375, 1.4541468620300293, 0.21556606888771057, 0.5902953743934631, -0.7224016785621643, 1.36764657497406, 2.2333791255950928, -0.22613534331321716, 0.07324077934026718, -0.9224085211753845, -0.49767830967903137, 0.8218078017234802, 0.45136016607284546, -0.12253645062446594, -0.17397135496139526, -0.2701978385448456, 1.0412566661834717, 0.3538835942745209, 1.0047823190689087, -0.176590234041214, 1.2291371822357178, -0.34674137830734253, 1.3652299642562866, 0.12090632319450378, -0.7937500476837158, -1.90941321849823, -0.7937694191932678, 0.22623777389526367, -0.773984968662262, -1.1155214309692383, -1.783313512802124, -0.7662214040756226, -2.077990770339966, -0.4144304096698761, -0.029231417924165726, -0.5014539957046509, 0.02228904515504837, -0.7530937194824219, -0.34324827790260315, 0.5301766395568848, 0.9853783845901489, 0.0033746869303286076, 0.23161640763282776, -0.0015309927985072136, 0.5172727704048157, -1.300060749053955, 2.6787729263305664, 1.122288465499878, 0.4700497090816498, 0.21653680503368378, -1.34603750705719, 1.1561486721038818, -0.8467074036598206, -0.8287832140922546, -0.5927228927612305, -2.175541400909424, 1.3649497032165527, 0.19241319596767426, -0.9644137024879456, -0.4917432963848114, 1.0524412393569946, -0.3641625940799713, -0.5868236422538757, 0.367851585149765, -0.9240897297859192, -0.9276381731033325, -1.0017268657684326, 0.45063281059265137, 0.6002012491226196, 0.5585382580757141, -0.2566365599632263, 0.01248872559517622, -0.5544136166572571, 0.28921952843666077, -0.6712510585784912, 0.2724578082561493, -2.190410614013672, 0.01752469316124916, -0.4536321759223938], [0.5836168527603149, -1.128154993057251, -1.1031049489974976, 0.8611152768135071, 0.6168377995491028, -0.6220623254776001, 1.1291083097457886, -0.057799357920885086, 0.24874292314052582, -1.565658450126648, 0.5176620483398438, -0.7409369945526123, 1.0973451137542725, 0.14070704579353333, 0.34468188881874084, 0.035642869770526886, 0.31200873851776123, -1.0571657419204712, -1.835781216621399, -0.2572956383228302, -0.1735038310289383, 1.0309518575668335, 0.6437858939170837, -0.8129041194915771, 2.536977529525757, -0.6571270823478699, -0.2830779254436493, 0.3456454575061798, -0.6039853096008301, 0.697558581829071, 1.9068235158920288, 0.04986194148659706, -0.8345446586608887, 0.08411543071269989, 0.2833521068096161, -2.1160366535186768, -0.7501056790351868, 1.1699466705322266, -1.2642186880111694, -1.4471911191940308, 0.9911054372787476, -0.7488645911216736, -0.624260425567627, 2.597153425216675, -1.3908474445343018, -1.8645055294036865, -0.2150196135044098, 0.6899915933609009, -0.5511293411254883, 0.3017388880252838, 1.1578071117401123, -0.14955948293209076, 0.0958755612373352, -0.9982715249061584, -0.035888414829969406, 0.6458829045295715, -1.0468171834945679, 0.316310852766037, -1.1925139427185059, 0.12371364235877991, 0.5310551524162292, -1.9143328666687012, 0.3155796527862549, -0.09641700237989426, 1.1105570793151855, 1.1594470739364624, 0.6542374491691589, 0.3084663152694702, -0.8263646960258484, 1.2889363765716553, 0.3444778323173523, -1.0139979124069214, -0.7841010093688965, -0.5214411616325378, 0.33817946910858154, 0.544275164604187, 0.3089476227760315, -0.7177550792694092, -0.8217539191246033, 0.4035602807998657, 0.22626732289791107, 0.029756879433989525, 0.11564314365386963, 0.7989474534988403, 0.4237666428089142, -1.292009711265564, -0.7478269338607788, -1.1581509113311768, -0.15521582961082458, -2.4033539295196533, -0.39492684602737427, -0.7159745097160339, -0.7611533999443054, 0.1269957721233368, 0.39518728852272034, 0.7960343956947327, -0.7001839876174927, 0.43360424041748047, 0.816220760345459, 0.3054371178150177, -0.2610296308994293, -0.9299876689910889, 0.11117035150527954, 0.7514256238937378, -1.0444847345352173, 1.1942135095596313, 0.9549906849861145, 0.2947550415992737, -0.9094716310501099, -0.7962592244148254, 1.6455363035202026, -1.7919328212738037, -2.3807015419006348, 1.3355345726013184, -0.5545850396156311, -0.4365355968475342, 1.53496515750885, -0.7318296432495117, -1.5211803913116455, -0.3492181897163391, -0.14350822567939758, 2.1447324752807617, 0.04356677830219269, -0.9341043829917908, -1.205772042274475, -0.20781981945037842, 0.9602448344230652, -1.1131161451339722], [-0.02232659049332142, 0.5492122173309326, -0.43870964646339417, 0.8698589205741882, -1.9724140167236328, -0.6881068348884583, 0.44065460562705994, 1.0206565856933594, -0.8193824291229248, 0.9520648121833801, -0.5163379311561584, 0.4479963779449463, 1.7136989831924438, -0.46340760588645935, -1.0240086317062378, -0.44610515236854553, 0.6273113489151001, 1.686420202255249, -1.0455392599105835, -0.8095199465751648, 0.4467152953147888, 0.7395512461662292, -1.5818357467651367, -0.4590337574481964, 0.2913369834423065, 2.042997360229492, -0.9043483138084412, -0.44358423352241516, 0.9129735231399536, 0.7177342176437378, 0.15446317195892334, 1.1900602579116821, 0.46441516280174255, -1.1652886867523193, -0.8952473402023315, -1.9147895574569702, -0.1295863389968872, 1.3957735300064087, -0.1467161774635315, -0.29881590604782104, 0.1925489902496338, 0.4503394663333893, -0.23656843602657318, 0.43253764510154724, -0.20351968705654144, -1.7456283569335938, 0.6311163306236267, 1.0942072868347168, 0.8366175293922424, 1.5515356063842773, -0.09819543361663818, -0.5840916037559509, 0.17768509685993195, -1.1270853281021118, 0.42934805154800415, -0.45232927799224854, -1.6147723197937012, -0.19351467490196228, -0.1913929283618927, -0.5523144006729126, 0.10291928797960281, 1.5649477243423462, -1.6419932842254639, -0.7398128509521484, -1.8020274639129639, -0.1258818358182907, -0.8143467307090759, -0.8412628769874573, 1.1364455223083496, -0.08276186138391495, 0.9486234784126282, 0.31883513927459717, -0.3714977204799652, 1.3638648986816406, -0.5845001339912415, 0.16047433018684387, -0.6468812227249146, -0.027242930606007576, -0.06925487518310547, -0.03524106368422508, 1.002297043800354, -0.0363704115152359, -1.8590689897537231, 1.0012985467910767, 0.0017429360887035728, 0.308217853307724, 0.6381757855415344, -2.6169285774230957, 1.6304210424423218, 0.6780731678009033, 1.9718588590621948, 0.3581024706363678, -0.8825951814651489, -0.8166766166687012, 1.0956008434295654, -0.27868661284446716, 1.8751530647277832, 0.06759238243103027, 1.1294212341308594, -0.4563477337360382, -1.2483094930648804, -0.8251011967658997, 0.5657841563224792, -1.6926994323730469, 1.0994805097579956, -1.1441571712493896, 1.6466877460479736, 0.5859740972518921, -1.2887197732925415, 0.4623083770275116, 0.6721899509429932, 0.15120211243629456, 0.3384908139705658, -0.32049304246902466, 0.5094918608665466, -0.3711431920528412, 0.7670607566833496, 0.4967474043369293, -1.2294062376022339, -1.6057430505752563, -0.7981876134872437, 0.056910742074251175, -0.8515138030052185, 1.7480943202972412, 0.30193355679512024, -0.7457902431488037, 1.3754632472991943, 0.485643208026886], [-1.7358664274215698, 0.9613970518112183, 0.20004653930664062, 1.1203564405441284, -0.1401224434375763, -1.1867263317108154, -0.5292438864707947, 1.6000005006790161, 1.0599347352981567, 0.71431565284729, 0.503566324710846, -1.6730622053146362, 0.16813834011554718, 1.3206868171691895, -1.0428543090820312, 0.0654694139957428, -0.43093374371528625, -1.4850995540618896, 1.3519482612609863, -1.9378045797348022, -0.30908411741256714, -0.06953292340040207, -0.08206508308649063, -1.6031763553619385, -1.1173244714736938, 0.06869403272867203, 0.9721238017082214, -0.6922109723091125, -0.09142600744962692, 0.2872047424316406, -1.9091638326644897, -1.6517441272735596, -0.8682170510292053, 1.4216806888580322, -1.0536713600158691, 0.6905356049537659, 1.506680965423584, -0.48872920870780945, 0.16721677780151367, -0.07475649565458298, -0.13277587294578552, -0.17858164012432098, -1.9126548767089844, 0.02047152817249298, 0.26799795031547546, 0.3098785877227783, -0.7923339009284973, 0.09559549391269684, -1.5289257764816284, -0.18129688501358032, -0.12907826900482178, -0.9856771230697632, 0.569858193397522, 0.2519271969795227, -1.0142396688461304, 0.2117399275302887, -0.48803696036338806, -1.1519324779510498, 0.4797273278236389, -0.8575888276100159, -0.4540443420410156, 2.91532301902771, 0.4530181586742401, -0.7044923305511475, -0.3215762972831726, 0.04879908263683319, 1.6995335817337036, 0.872649610042572, 0.22380699217319489, -0.56122887134552, 0.3617127537727356, 1.2959367036819458, 0.6485310196876526, 0.06130469590425491, -1.3389288187026978, -1.4534132480621338, -1.387777328491211, -0.1858081817626953, 1.8031890392303467, -0.5813176035881042, -0.6218902468681335, -1.3475415706634521, -1.4751827716827393, -0.9167149066925049, 0.3566172420978546, 0.3803144097328186, -0.018596511334180832, -1.086320400238037, -1.7089216709136963, -1.1346313953399658, 1.1531949043273926, 1.27532958984375, 0.32501181960105896, 0.6132850646972656, -1.0255334377288818, 0.4476202428340912, 0.551619827747345, 0.7523748278617859, 1.214919090270996, -0.6092587113380432, -0.09021572023630142, -1.6016716957092285, -1.9227007627487183, 1.2154951095581055, 1.5411919355392456, -1.3123278617858887, -0.24867410957813263, 0.3190152645111084, 0.19636167585849762, -0.405407190322876, -1.8701461553573608, -0.4443206489086151, 0.9571977257728577, -0.8197428584098816, 0.01678316481411457, 0.15976107120513916, -0.4919643998146057, 0.8834848999977112, 0.7881222367286682, -0.5022071599960327, -0.7435289025306702, -0.7577045559883118, -0.4193103313446045, -0.590488076210022, 1.810073733329773, 0.14964495599269867, -1.3177345991134644, -0.1084037497639656], [0.36108529567718506, 0.49060818552970886, 1.673248529434204, -0.2863500416278839, 0.38653501868247986, 1.7164063453674316, -1.5051066875457764, 0.09227596968412399, -0.6782965660095215, -0.9652935862541199, 0.5203744769096375, -0.7855282425880432, -0.8267510533332825, 0.1152765154838562, -1.2737101316452026, 2.3772969245910645, -0.49476301670074463, 0.9620822668075562, 1.045668601989746, -0.45049792528152466, 1.4263200759887695, -1.7328717708587646, 1.8884859085083008, 0.4238995909690857, -0.6036590933799744, -0.5928041934967041, -0.9689135551452637, -0.6534660458564758, 0.2974030077457428, -0.7835081815719604, 0.08263610303401947, 0.18747319281101227, 0.5306863784790039, -0.468578577041626, 0.2658470869064331, -0.9600158333778381, -1.3261915445327759, -0.5470330715179443, -0.08230407536029816, -1.41907799243927, -0.3947370946407318, 0.5698884129524231, 0.8574933409690857, 2.0856053829193115, -0.7050825953483582, -1.7092159986495972, 1.4847948551177979, -1.146223783493042, -0.7023658156394958, -0.026363801211118698, 1.2040619850158691, 1.3359256982803345, 1.1168479919433594, 0.2275322824716568, -1.487015724182129, -0.5269877314567566, 0.30397456884384155, 0.3139207363128662, -0.40222084522247314, 2.113231658935547, 0.19651848077774048, -0.13237743079662323, 0.4983868896961212, -0.1736314296722412, 0.5942854881286621, -0.38380393385887146, -0.7254018187522888, -0.13706117868423462, -1.804659366607666, -0.44980454444885254, 0.6575565338134766, 0.0649174377322197, 1.3752775192260742, 1.5122277736663818, 0.9859304428100586, 0.5321592092514038, 0.7797554135322571, -0.3663473427295685, 1.7288569211959839, 1.0535062551498413, 0.18194067478179932, 0.5397130250930786, 0.8983454704284668, 1.2475714683532715, -0.14193890988826752, 0.36595410108566284, 1.748548150062561, -2.0022120475769043, 0.25483208894729614, 0.24585427343845367, 0.35061535239219666, -1.1719890832901, -0.35524508357048035, 0.6058409810066223, -1.1765722036361694, -0.12222246825695038, -0.745930016040802, 0.8036309480667114, 0.14264826476573944, 0.8878483176231384, 2.4365921020507812, -0.8330562710762024, 0.38316094875335693, -1.7218798398971558, 0.1117083728313446, 1.3347587585449219, -0.4001356065273285, 0.08779699355363846, 0.223411425948143, 0.7560117244720459, -0.29585474729537964, -0.8203819990158081, -1.5100669860839844, -0.540627121925354, -1.663906455039978, 0.9813316464424133, 2.4901671409606934, -0.27433422207832336, 0.8810932636260986, -0.7397192120552063, 0.14644484221935272, -1.4612098932266235, -0.9440596699714661, -0.7007239460945129, -1.6931567192077637, -1.367805004119873, 0.4048422873020172, -0.36347752809524536], [-0.4131752848625183, 1.149857759475708, 0.6150870323181152, -0.41513592004776, -0.35877999663352966, 0.4958529472351074, 0.3803645968437195, -0.434680312871933, -2.1030261516571045, 0.16144774854183197, -0.2322196364402771, -0.08434635400772095, 1.1518265008926392, 0.39064154028892517, -0.2074493020772934, -0.29121968150138855, 0.2435038536787033, 0.4766187369823456, 0.45489975810050964, -0.3780326843261719, -0.6195741891860962, -0.6181633472442627, 1.4620232582092285, -1.0890475511550903, -0.5676579475402832, -0.8583394885063171, -0.15087412297725677, 0.7098535299301147, 1.3849546909332275, 0.9939587712287903, 0.5934526920318604, 1.090602159500122, -1.3981060981750488, -0.3953050374984741, -1.471052646636963, -0.560102105140686, 1.847769021987915, 2.6869375705718994, 0.13261330127716064, 1.2771668434143066, -0.7477888464927673, 0.6802242398262024, 1.1894397735595703, 0.4827285706996918, 2.4528093338012695, 0.7839869260787964, -0.8460619449615479, -2.023075819015503, 0.2447407841682434, -1.5974504947662354, -2.061551332473755, -1.012117862701416, 0.4408528804779053, 0.4258333444595337, 0.604849100112915, 0.19716301560401917, -1.2466721534729004, 2.3821609020233154, -0.26516154408454895, -0.9373996257781982, 0.9189923405647278, 1.9844634532928467, -0.27306246757507324, -1.1262305974960327, 0.5651792883872986, -1.5009175539016724, 0.06044352054595947, -1.4720538854599, -2.0030126571655273, -0.9687616229057312, 1.027001976966858, 1.8902260065078735, -0.04692119359970093, 1.054783821105957, -0.03792066499590874, -0.5090529322624207, 0.49077877402305603, -0.22986476123332977, -0.15477712452411652, -0.7282716035842896, 2.38210129737854, -0.05638781189918518, -0.7760895490646362, 0.4110625088214874, 0.3621416389942169, 0.8660764694213867, -2.0390372276306152, -0.7653892040252686, -0.008614715188741684, -0.0850643739104271, -0.32091024518013, -0.6636326909065247, -1.469576358795166, -2.0240895748138428, 0.08999020606279373, -0.8509358763694763, -0.2465633749961853, 1.0784636735916138, -0.503360390663147, -1.9154765605926514, -1.6729971170425415, -1.176041841506958, -1.4668861627578735, 1.088753342628479, -1.0974931716918945, 1.362883448600769, 0.45352450013160706, 0.5560451149940491, 4.281327724456787, 0.9223714470863342, -0.7717582583427429, -0.7630108594894409, 0.7966611981391907, 1.5838226079940796, 0.015580396167933941, -0.22153392434120178, -0.3614543676376343, 0.7364996671676636, 0.08006581664085388, 1.0173002481460571, 0.9560282826423645, 2.563400983810425, -0.2785210609436035, -0.9117327332496643, 0.46172258257865906, -0.0966092050075531, 0.3352668881416321, 0.09909922629594803], [0.5711860656738281, -1.7721078395843506, -0.7031840085983276, 1.2098642587661743, 0.17450714111328125, -0.44580787420272827, 0.45195427536964417, 0.0002872189215850085, 1.6456149816513062, -1.9784095287322998, 1.3087162971496582, 0.0028875565622001886, 0.29116740822792053, -0.6672147512435913, 0.4104992151260376, -1.8803534507751465, 0.27370432019233704, -0.339764803647995, -1.4362326860427856, -0.8520123362541199, 1.1737074851989746, -0.3708323538303375, -0.8292973637580872, 2.16629695892334, 0.3318958282470703, 0.9130294322967529, 0.6483401656150818, 0.6331278085708618, 0.6278990507125854, -0.22972813248634338, -0.37336623668670654, -1.9514042139053345, 1.175121545791626, -0.3971931040287018, -0.8093805313110352, -1.8339735269546509, -0.6917242407798767, -0.23870491981506348, 1.7139009237289429, -1.4713026285171509, 1.0772768259048462, -0.22456026077270508, -0.5165987610816956, -0.4101913273334503, 0.1944994181394577, -1.5594438314437866, -0.06231915205717087, -0.6612727046012878, 0.6168737411499023, 0.48854026198387146, 1.1140456199645996, 0.8122462034225464, 0.170523539185524, 1.0139366388320923, 0.29426735639572144, -1.3327751159667969, -1.165544033050537, -1.478564977645874, 0.2552398443222046, 1.3302611112594604, -0.3121016323566437, -1.7823357582092285, 0.18119069933891296, 1.3441096544265747, 0.09647859632968903, 0.45221906900405884, -0.4436382055282593, -1.2821873426437378, 0.07430703938007355, 0.34887179732322693, -0.6124049425125122, 0.04475967958569527, 0.6926446557044983, 0.09230564534664154, 1.024398922920227, 1.2297877073287964, 1.6451804637908936, -1.0693705081939697, 2.6097066402435303, 0.19108489155769348, 0.26362964510917664, 0.4125956892967224, -2.272414445877075, -1.5350697040557861, -0.8081147074699402, 0.017084501683712006, -1.0184516906738281, 0.7974735498428345, -1.957371473312378, 0.6163715124130249, -1.1640214920043945, 0.010232930071651936, -0.49252644181251526, -0.5408584475517273, 0.11187315732240677, -0.9686604142189026, 0.6840080618858337, -0.4419757127761841, 0.4405241906642914, 1.0256297588348389, -0.2284557968378067, 0.5371611714363098, 1.039677381515503, -0.7839885354042053, 0.570958137512207, 0.2769690454006195, 1.2703746557235718, 0.4154297709465027, -0.12052372097969055, 1.7206835746765137, -0.881895899772644, 0.4878285229206085, 2.1084277629852295, -0.9183589816093445, -1.269431471824646, -0.48702022433280945, 0.5763892531394958, -0.20751290023326874, -1.2112212181091309, 0.7766714096069336, 0.03925824537873268, -0.17255601286888123, -0.3901045620441437, 0.6546862125396729, -1.1442482471466064, 0.3106462359428406, -1.205079197883606, -0.6410843133926392], [0.1200333684682846, -0.6118667721748352, 0.08373622596263885, 1.0542627573013306, -0.8648779988288879, 0.2575826048851013, -1.6113141775131226, -0.24820546805858612, 0.5047421455383301, -1.6492441892623901, -1.7538777589797974, -0.1086413562297821, 0.01640067994594574, 1.5819658041000366, -0.7190134525299072, -0.21362215280532837, -1.4741809368133545, 0.32209548354148865, 0.09675177931785583, -0.6193909049034119, -2.1475744247436523, -1.453262209892273, -0.44714540243148804, 1.119722843170166, -0.12588749825954437, -0.08061880618333817, 0.40183842182159424, -0.7536364197731018, -0.3830912411212921, -1.1198238134384155, 0.5393616557121277, -1.9687491655349731, -1.1116642951965332, -0.12495908886194229, -0.3734111785888672, 1.004995584487915, -1.135738492012024, -0.585446834564209, -0.44423624873161316, -0.8700150847434998, 1.5265752077102661, -0.36361873149871826, 0.7603497505187988, -1.6978046894073486, 1.3385465145111084, 0.5545929670333862, 0.32534059882164, -1.4695281982421875, 0.5900099873542786, 0.18440808355808258, 1.0484187602996826, 2.4318790435791016, 0.34530115127563477, 1.0912786722183228, -1.0575023889541626, -1.62277352809906, 0.7303787469863892, 0.4597598612308502, 0.0843803882598877, 0.5370280146598816, 1.5622907876968384, -0.7496780753135681, 0.20086054503917694, -0.9663623571395874, -0.4174153804779053, -0.9478012323379517, -0.4329679608345032, 0.47176027297973633, 0.25327372550964355, -0.11184454709291458, -0.28293293714523315, 0.566745936870575, 0.2750076651573181, 0.17239049077033997, 0.7101423740386963, 2.2508347034454346, 0.501803457736969, -1.047392725944519, -0.038662537932395935, 0.3871432840824127, -0.7568838596343994, -1.0062235593795776, 1.1552467346191406, -0.971410870552063, -1.4096431732177734, -0.23661920428276062, -0.3694871962070465, -2.2273552417755127, -0.2586001455783844, 0.009799408726394176, 0.3579701781272888, -0.8885435461997986, -0.7345865368843079, -0.5291603207588196, -1.0094283819198608, 0.6434112787246704, -0.9286400079727173, 0.785271167755127, -0.818156898021698, -1.125831127166748, 1.218975305557251, -0.39321690797805786, -1.6684712171554565, 0.10261324793100357, 1.3118281364440918, 1.3106364011764526, -0.32777532935142517, -0.928784966468811, -0.0379442498087883, 0.4560682773590088, 0.40618354082107544, -0.6391202211380005, -0.8827879428863525, -0.8432422280311584, 0.49994033575057983, 1.0537104606628418, 1.708987832069397, -0.42362216114997864, 0.18065804243087769, 0.2870109975337982, -0.8811005353927612, 0.3806893527507782, 2.0902774333953857, -0.8560421466827393, -1.4638919830322266, 1.3688504695892334, 0.5940128564834595, 1.049311637878418], [-2.1618452072143555, 0.2943463623523712, 0.5469056963920593, 1.4165723323822021, 2.25289249420166, -0.6771288514137268, -1.1225571632385254, -1.2316415309906006, -1.2754539251327515, -1.163430094718933, 0.12444424629211426, 0.5637683272361755, 2.562549591064453, -1.991459846496582, -0.1024840772151947, 0.2703397870063782, 0.10454101860523224, -1.1472219228744507, 1.4522989988327026, -1.1045093536376953, 0.4675340950489044, 0.26090994477272034, -0.8047597408294678, 1.0643342733383179, -0.30578333139419556, -1.8577090501785278, 0.4661499261856079, -0.4990233778953552, 1.4631210565567017, 0.18526311218738556, 0.34530240297317505, -1.28499174118042, 1.1273730993270874, -0.008060397580265999, -2.410706043243408, 0.34797829389572144, 0.8127980828285217, -0.1692570447921753, -0.03642743080854416, -0.938896119594574, 1.1527206897735596, -0.20835396647453308, 0.1323864459991455, 0.46551674604415894, 2.4236042499542236, 0.10395902395248413, 0.6022834777832031, 0.49573346972465515, -0.4882145822048187, -0.24676913022994995, 0.31237074732780457, -1.0560663938522339, -0.5044716596603394, 0.19214949011802673, 0.6985818147659302, -1.0409307479858398, -2.3439388275146484, 1.0537959337234497, -0.6869436502456665, -0.43975844979286194, -0.3158901631832123, -1.841516137123108, 1.3318660259246826, -0.6660462617874146, -0.7050510048866272, -1.1141310930252075, 0.1027405634522438, 0.1871212273836136, -0.09646707028150558, -0.7538343667984009, -0.10414943844079971, 0.3596024215221405, -0.4283563196659088, 1.6093673706054688, -1.897183895111084, 0.824104905128479, -1.4828541278839111, -1.4009534120559692, -0.8529163002967834, -0.2226434350013733, 1.7469931840896606, 1.2868642807006836, -0.5545173287391663, -0.27211084961891174, 0.9811763167381287, -1.090315341949463, -0.7016432881355286, 0.7659969329833984, -0.7251871824264526, -1.4155765771865845, 1.343217134475708, -0.9247501492500305, -0.2188810408115387, -0.9373207688331604, 0.5839827656745911, 1.2953914403915405, -0.4308485984802246, 1.6894440650939941, -0.8341363072395325, -1.1229513883590698, -0.497365266084671, 0.6122172474861145, -0.14670224487781525, 0.154075026512146, 0.6280229687690735, -0.7128350138664246, -1.7929821014404297, -0.10835538059473038, -0.04837416857481003, 0.4859878122806549, 0.6064451932907104, 0.1745833158493042, 1.269078254699707, -1.051708459854126, -0.5035620331764221, -1.0148835182189941, -0.6509049534797668, 1.1250778436660767, 0.2276574671268463, 0.5869459509849548, 0.3268173635005951, 0.6123296022415161, 1.0660426616668701, 0.5592012405395508, 0.1836043894290924, -0.12428824603557587, 1.1998252868652344, -0.4504694640636444], [-0.37267187237739563, 1.8393280506134033, 0.3736141324043274, 0.7073833346366882, -1.7479914426803589, 0.2859545946121216, -0.38442516326904297, 0.005071087274700403, -0.6294834613800049, -1.5497397184371948, 0.8113913536071777, -0.7326053380966187, 0.04056797921657562, 0.22084252536296844, 1.1022601127624512, 0.17364241182804108, 0.9053553938865662, 0.5279042720794678, -0.5011485815048218, -1.3098465204238892, 0.7023482918739319, -0.6114794015884399, -0.08511195331811905, 0.3985387980937958, -0.6207908391952515, -2.504018783569336, 1.2381229400634766, -0.39424338936805725, -1.0470184087753296, 2.0567784309387207, -0.698154628276825, 0.903168797492981, -0.7103316187858582, -0.09143266826868057, -0.4890514016151428, -1.747763991355896, 0.09875009208917618, 0.3064643442630768, -0.2487364411354065, 0.44678783416748047, -1.1616699695587158, -0.02032547816634178, -0.12452185153961182, 1.315738558769226, 0.3589601516723633, -0.11187997460365295, 0.9147673845291138, -0.26658934354782104, -0.2666361331939697, -1.2768949270248413, -0.6114610433578491, 1.3537579774856567, 0.8036624789237976, 0.1612207442522049, -0.9639586806297302, -0.32266584038734436, 1.5867695808410645, 1.096129298210144, -0.2000686228275299, 0.5218972563743591, 0.9648618102073669, 0.7234771847724915, 0.36908215284347534, -1.2264404296875, 0.5821656584739685, 1.0529627799987793, -0.5551102161407471, -0.9850900173187256, 0.8447071313858032, 0.24604861438274384, -1.2588670253753662, -0.733513593673706, -0.560303807258606, -0.1551855355501175, 0.11106453090906143, -0.9069989323616028, 0.7797877192497253, 1.9004536867141724, -1.7101545333862305, 0.44796159863471985, 0.3287681043148041, 0.11739297211170197, -1.635632038116455, 0.22119610011577606, 0.3673020899295807, -0.04214920848608017, -0.18923884630203247, -0.033568982034921646, 0.3552871644496918, 1.0781453847885132, -0.08764390647411346, -0.7675920724868774, -0.609124481678009, 1.5375452041625977, 1.787524700164795, 0.21896880865097046, -0.5908558964729309, -0.0018691284349188209, -1.144275426864624, -0.48854973912239075, 0.10210905969142914, 1.4664252996444702, -0.6482673287391663, 0.8786709308624268, 1.0696470737457275, 0.9310802817344666, -1.3519822359085083, -0.6445575952529907, 0.5659629702568054, 0.7638929486274719, -0.36262860894203186, 0.09205850213766098, -0.523165762424469, -0.26058629155158997, 0.023233095183968544, -0.6541089415550232, 1.9593946933746338, 0.051631368696689606, -0.21110598742961884, 0.5600379705429077, -0.1588316559791565, -1.0568493604660034, -0.022205233573913574, -0.9207631945610046, -0.5931490659713745, 0.43010690808296204, -0.5460665822029114, 0.7342853546142578], [0.16583633422851562, 0.4476228356361389, 0.5075432062149048, -0.23515501618385315, 0.01804172806441784, -0.3044077455997467, -0.4557139575481415, 0.11632738262414932, -1.3204041719436646, -1.27154541015625, -0.5665408372879028, 1.2854323387145996, -0.39314863085746765, -1.690415620803833, 0.5200285911560059, -1.366096019744873, -1.5243096351623535, 0.012692691758275032, -0.2963091731071472, 0.7039098739624023, -1.375807285308838, -1.7043570280075073, 1.7417242527008057, 1.1873724460601807, 1.181105375289917, 1.1520617008209229, -0.26742538809776306, -1.8122977018356323, 0.24266639351844788, 1.4819064140319824, 0.409538596868515, 1.1294571161270142, 2.3355302810668945, -0.23626267910003662, -0.9677813053131104, 1.5033321380615234, 1.2899737358093262, -0.73458331823349, 2.002635955810547, -0.17831192910671234, -0.4732655882835388, 0.8676438927650452, -0.27969998121261597, -0.9633967280387878, 0.6476088762283325, -0.2029375433921814, 0.3304762840270996, -0.7859506607055664, 1.0866646766662598, 0.7535648345947266, -0.6665419340133667, -0.37398603558540344, 0.09992137551307678, -0.10151206701993942, 1.1576002836227417, 1.7149670124053955, 0.5644971132278442, 0.5512511134147644, 3.1734132766723633, -1.2162271738052368, -0.729638397693634, -2.002729654312134, -0.7026578187942505, 0.4457698464393616, -1.6396727561950684, -0.24002696573734283, -0.6959059238433838, 0.47430577874183655, -1.71418035030365, 0.274544358253479, -1.5649347305297852, 0.5678244233131409, -1.4481821060180664, 1.1046233177185059, 0.18273548781871796, -0.3547581434249878, 0.10352209955453873, -2.508904457092285, -2.338933229446411, 0.09273336827754974, 0.41247886419296265, -0.6220442056655884, -0.6399856805801392, -0.43552452325820923, -0.43342795968055725, -0.5883519053459167, 1.004604458808899, -0.05647886171936989, -0.07529299706220627, 0.3422999083995819, 1.3826310634613037, 1.5237501859664917, -1.2538011074066162, -1.5589613914489746, 0.6706879734992981, 0.2592112720012665, 0.48269030451774597, 0.2885397970676422, -1.1187742948532104, -0.2702002227306366, -1.211061716079712, -2.238562822341919, -0.28500136733055115, 1.5636128187179565, 1.07741117477417, 1.4513099193572998, 0.566331684589386, -0.2198483645915985, 0.045831695199012756, 0.41029444336891174, -1.258497714996338, -0.4296018183231354, 0.4080638885498047, 2.096266269683838, -1.1193195581436157, -0.12756548821926117, 0.3173481225967407, -1.762343406677246, -0.8567941784858704, -0.22172191739082336, -0.21620050072669983, 0.535780131816864, -2.401257038116455, -1.0946494340896606, -0.8711017966270447, -1.4854357242584229, -0.6755560040473938, 1.7261074781417847], [-1.7954481840133667, 1.484349250793457, -1.731064796447754, 1.944510817527771, -0.8788377642631531, 2.783632278442383, 0.8317379355430603, -1.6114634275436401, 0.8840659856796265, 0.09935358911752701, -1.544273853302002, 0.9849055409431458, -0.29492712020874023, -1.4134743213653564, 0.3879120945930481, -0.23284146189689636, -0.7079224586486816, 0.3473292589187622, -1.247928500175476, 0.43497875332832336, 1.0672845840454102, 0.10821634531021118, 2.226665735244751, 0.17678852379322052, 1.0077334642410278, 1.0629805326461792, -1.4453092813491821, -0.9779694080352783, -1.9838322401046753, -0.5054055452346802, -0.37722787261009216, 0.10907494276762009, -2.426767349243164, -0.28178250789642334, -0.8240077495574951, -0.5962603092193604, 0.8749812841415405, 0.843757688999176, -0.13762140274047852, 0.1489446759223938, 1.2180569171905518, -0.36112645268440247, 0.6789577007293701, 0.035035617649555206, 1.334551453590393, -0.4341966509819031, 0.3589257299900055, 0.7777249217033386, -0.29972103238105774, 1.029524326324463, 0.5262618064880371, 0.24893847107887268, 0.6851567625999451, 0.3247435986995697, 0.8355205655097961, -1.5044865608215332, -1.3307371139526367, 0.3704943060874939, -0.5190008282661438, 0.5805631875991821, 0.4342264235019684, -0.2853703200817108, 0.18451383709907532, -1.4437816143035889, 0.4267735183238983, 0.9505433440208435, -0.7418972253799438, 0.9396001100540161, -0.6861616969108582, -1.6609537601470947, 0.022011753171682358, 1.2651665210723877, 0.174312025308609, -1.1641511917114258, 0.4113091826438904, 0.4330686032772064, -1.2991317510604858, -1.761125087738037, 0.2979820668697357, -0.9777047038078308, -0.7807621955871582, 0.24090039730072021, -0.8015056848526001, 0.038906216621398926, -0.4077395498752594, -1.3431397676467896, 0.14800220727920532, 0.3063068389892578, -1.9421182870864868, -1.100992202758789, -0.12168679386377335, 0.3859602212905884, -0.5777100324630737, -1.0444597005844116, 0.2640382647514343, -0.8292285799980164, -0.32613736391067505, -0.014982704073190689, 0.49346083402633667, -0.6712695360183716, 0.21946987509727478, -0.19988319277763367, -0.9036881923675537, 0.5271732807159424, 0.025603584945201874, -0.39737579226493835, -1.4462600946426392, -0.034698486328125, 0.276328980922699, 0.8162899613380432, 0.17737315595149994, 1.5895769596099854, -0.4856996536254883, 0.9883702397346497, -0.659398078918457, 1.3621885776519775, 0.599221408367157, -0.475927472114563, -2.514979839324951, -0.0683094784617424, -0.8127104640007019, -0.4327841103076935, -0.5805396437644958, 1.0691545009613037, -0.930600106716156, 1.4355322122573853, 0.5997164249420166, -1.6947622299194336], [-1.6448317766189575, 0.8711004853248596, 1.8152532577514648, 1.2426226139068604, -1.241809606552124, 1.1296740770339966, -0.5218451619148254, 0.7656865119934082, -0.8984413146972656, -1.1068320274353027, 0.2720145285129547, 1.6362982988357544, 0.9643903970718384, 0.8196263313293457, 2.1570756435394287, 0.5260711908340454, 0.4794714152812958, 0.4574519991874695, 0.17878803610801697, 1.4471235275268555, -0.6962204575538635, 0.5470154285430908, 0.681045413017273, -1.2263966798782349, 0.01100081205368042, -0.531950831413269, 0.5642398595809937, -0.7723190784454346, 1.2184466123580933, 1.7843027114868164, 0.25750935077667236, -1.3220489025115967, 0.8870612978935242, 0.9901749491691589, 0.4947011172771454, 0.22047092020511627, 0.1773741990327835, 1.2250560522079468, -0.45341789722442627, 0.7658697366714478, -0.006215389352291822, 0.2731979787349701, -0.1645621508359909, -1.4614222049713135, 1.7227871417999268, 0.17959082126617432, 0.1590558886528015, -1.3255044221878052, -0.27772197127342224, 0.28284475207328796, -0.8711877465248108, -0.7936233878135681, 0.5063572525978088, 0.5917264819145203, -1.0199202299118042, 0.6726024746894836, -0.8821712732315063, -1.5115028619766235, 0.5107381343841553, -2.089816093444824, 0.7826557755470276, -1.8727298974990845, -0.001632586121559143, -0.36851659417152405, 0.3719587028026581, 0.43365058302879333, -0.6397750377655029, 0.2164081335067749, -1.1951643228530884, 0.6240396499633789, -0.7531936168670654, 0.7775339484214783, -1.8948116302490234, -0.8860912322998047, -0.2701469361782074, 1.1168385744094849, -0.24361126124858856, 1.3312500715255737, -1.4069278240203857, 1.3831536769866943, -0.9099510312080383, 0.5857406258583069, 1.656286597251892, 0.6024953722953796, 0.6842851638793945, 0.5398894548416138, -1.481462836265564, 0.763550341129303, -1.938615322113037, -0.8213969469070435, -0.8470432758331299, 0.4416755139827728, 0.13882699608802795, 0.5740976333618164, -0.4482124447822571, -0.9948680400848389, 1.0089143514633179, -1.11069917678833, -0.34287896752357483, -1.7976638078689575, -1.8251415491104126, -0.7969317436218262, -1.4947608709335327, -1.5377637147903442, -1.5594978332519531, -0.8646650910377502, -1.2100920677185059, -1.5357508659362793, 0.22983041405677795, 0.9247242212295532, -1.4166226387023926, -1.6589624881744385, -1.7685104608535767, -0.5495559573173523, 0.3888324797153473, 1.0665513277053833, 2.198997735977173, 1.4663231372833252, 0.7887555360794067, 0.1388864815235138, -0.20529448986053467, -0.7977481484413147, 0.8431811332702637, -0.8703880310058594, 2.0440452098846436, -0.07207255810499191, -0.6588088274002075, -0.7864766716957092], [1.0297502279281616, -0.30690184235572815, 1.263581395149231, -2.649973154067993, 1.0585540533065796, 2.5762410163879395, -0.23772314190864563, 1.0623403787612915, 0.006480591371655464, 0.5630622506141663, 0.27836889028549194, 1.474621295928955, -1.0528895854949951, 0.6974648237228394, 0.01795138418674469, -1.3544548749923706, 0.09050396829843521, -0.7247222065925598, -0.7041939496994019, -0.2627622187137604, -0.07845033705234528, -0.7125886678695679, -1.5278825759887695, -1.356559157371521, -0.11552809923887253, 1.8368878364562988, -0.8339728116989136, -0.7691663503646851, -1.6323742866516113, 0.20166118443012238, 0.31473949551582336, 0.2987031042575836, -2.119948148727417, -1.0418941974639893, 0.4533363878726959, 0.8986366391181946, -0.7187134623527527, 0.8966353535652161, 0.26874199509620667, -1.0705204010009766, 1.207679033279419, 0.1234489306807518, -2.096086025238037, 1.1982649564743042, -0.205524742603302, -0.6171988844871521, 1.3362159729003906, 1.0756216049194336, 1.2907227277755737, 0.7619083523750305, 0.39278364181518555, -0.484437495470047, -1.1784470081329346, -0.0534915030002594, 1.048693060874939, 0.24792315065860748, -1.0814133882522583, 1.5557745695114136, 1.4859179258346558, 0.1414494514465332, -2.6611156463623047, -0.48399531841278076, -1.194022297859192, 0.05345407873392105, -0.16758087277412415, -0.19401049613952637, 0.5353710651397705, -0.27574265003204346, 0.9189213514328003, 0.8164797425270081, 1.2224749326705933, 2.1713509559631348, -1.8705865144729614, 0.9913860559463501, -1.1639670133590698, 1.0833443403244019, -0.6254628300666809, 1.7258400917053223, 0.6313490271568298, 1.285705804824829, -0.7943600416183472, -0.8970984816551208, -1.45078706741333, 0.6565437912940979, -0.13054189085960388, 1.1105871200561523, 0.8401727676391602, 0.06154422089457512, 2.0075247287750244, -1.145590901374817, -0.7088639140129089, 1.0606430768966675, -0.6472135782241821, 0.540597140789032, 2.1123316287994385, 0.8162712454795837, 0.09757571667432785, -0.8712019324302673, -0.6961847543716431, -0.08916369080543518, 1.236283302307129, -1.8662700653076172, 0.44024139642715454, -0.5149282217025757, 0.9446626305580139, -0.09523043781518936, -0.8109731674194336, 0.23022519052028656, -0.9413864612579346, 0.11863959580659866, -1.0435181856155396, -1.3895052671432495, -0.6734483242034912, 0.051947787404060364, 0.3303409218788147, 0.6049185395240784, -2.0458126068115234, -0.25258851051330566, 0.5165484547615051, 0.8678200244903564, -1.5911628007888794, -0.4861639738082886, -0.37664738297462463, -0.9691145420074463, 1.875953197479248, 0.7211506962776184, 0.4722164273262024, -1.8468756675720215], [0.6983855366706848, -1.773795485496521, -1.102272391319275, -0.05063871666789055, -2.1079511642456055, 0.1775081753730774, 0.3015819787979126, 0.5963051319122314, -0.34622055292129517, 0.13795918226242065, 0.22056007385253906, -0.29607802629470825, -2.2593846321105957, 0.2967822253704071, -1.9867676496505737, -0.9749146699905396, 1.3616362810134888, 0.08555857092142105, 1.6928821802139282, 2.2391109466552734, -1.462555170059204, 0.21434049308300018, 1.7587765455245972, -2.93349289894104, -1.1465529203414917, 0.10088817775249481, -0.9460089802742004, 1.4177701473236084, 0.5216280817985535, 1.3929113149642944, 0.10581545531749725, 0.8680790662765503, 2.2787413597106934, 0.397267609834671, -1.0093309879302979, 0.4340526759624481, 0.5429618954658508, 0.4879993498325348, 0.393555223941803, 0.05658577010035515, -0.15608206391334534, -0.39440515637397766, -0.022593192756175995, -0.3242345154285431, 0.579003095626831, -2.3767433166503906, 1.359476923942566, 1.2047172784805298, 0.4489535689353943, 0.7981519103050232, 0.07446049898862839, -0.11137095093727112, -1.6157876253128052, -0.8915795683860779, 1.5167536735534668, 0.41482216119766235, -0.8959594964981079, 0.029864108189940453, -2.1079585552215576, -0.4994083642959595, 0.8349084258079529, -0.6961511969566345, -3.4358737468719482, 1.3000049591064453, -0.3720097839832306, 0.9751570820808411, 0.06305947154760361, -1.3925068378448486, 0.9776986241340637, 1.772337555885315, 1.1995370388031006, 0.2762604355812073, -0.4607456624507904, -1.2603355646133423, 0.8960277438163757, 1.881536841392517, -0.15215598046779633, -0.516228973865509, 0.34396862983703613, 0.48713386058807373, 0.7081807851791382, -1.2748955488204956, -0.6735215187072754, -0.8064645528793335, 0.6209559440612793, -1.7099319696426392, -2.641761064529419, 1.4172697067260742, 1.150003433227539, -1.7608754634857178, -0.2033485323190689, -0.09349977970123291, 0.9215664267539978, -0.1835441291332245, -0.4783468246459961, -0.5530277490615845, 2.0487022399902344, -0.08447069674730301, -0.1684122383594513, 0.5357969999313354, -0.3010214865207672, 0.2623792886734009, -0.2354200780391693, -0.18250082433223724, 0.7408400774002075, -0.12435707449913025, 0.546542227268219, 0.6160657405853271, 0.1020456925034523, 0.5658623576164246, 2.86910343170166, 2.0804452896118164, 0.667079746723175, -0.030341850593686104, -0.8214719891548157, -1.4873794317245483, -0.9718638062477112, -0.7045350670814514, -0.9128066301345825, -0.707693874835968, 0.4426110088825226, -0.34053272008895874, -2.304699182510376, 1.7003724575042725, -0.4278653860092163, -0.2472468912601471, -1.1761068105697632, 0.6534037590026855], [-0.556080162525177, 0.474718302488327, -0.04199221730232239, -0.9685903787612915, 0.33776623010635376, -0.6573618650436401, 1.9013781547546387, 0.9479507803916931, 1.1738008260726929, 0.4157063961029053, 0.4049188196659088, 0.035217199474573135, 2.480201244354248, 0.823961615562439, 0.20520544052124023, 0.18313471972942352, 1.9012730121612549, -1.6368542909622192, 0.6269956231117249, 1.4069105386734009, 1.2080175876617432, -2.3228673934936523, 0.9153648614883423, 0.9360466003417969, 2.956745147705078, 1.4145017862319946, -0.7820324301719666, 0.537555456161499, 0.2522808313369751, 0.6365827322006226, 0.6531099081039429, -1.2813575267791748, -0.03236155956983566, -0.07130325585603714, 0.720162034034729, -1.5168644189834595, -0.6487494111061096, 0.571696400642395, 2.169973611831665, 1.4503076076507568, -0.19535857439041138, -1.2531198263168335, -0.388688862323761, -1.6833897829055786, -0.06596604734659195, -0.20024292171001434, 0.916441798210144, -1.1474241018295288, -0.720810055732727, -0.657457172870636, -0.773184061050415, 1.4307478666305542, 1.3003509044647217, -0.16163098812103271, -0.6642915606498718, 1.521630883216858, 1.550744891166687, -1.1190375089645386, 0.23973150551319122, -0.12649030983448029, -0.05111226439476013, -0.08748587965965271, -1.4166547060012817, -0.11745691299438477, -0.5173920392990112, 0.4587901830673218, -1.0189573764801025, -0.9576273560523987, -1.9093701839447021, 0.4981541037559509, -1.0605437755584717, -0.1831260621547699, -0.09678048640489578, 0.754146158695221, -0.31938254833221436, 0.038363151252269745, -0.2957373857498169, -0.25850483775138855, 0.06972935795783997, 0.13142867386341095, -0.6368641257286072, 0.08833593130111694, -1.0075088739395142, -0.534321129322052, 0.5889844298362732, -0.2002801150083542, -0.4107261002063751, -1.7016221284866333, 0.30869948863983154, -0.34066614508628845, 0.607323408126831, 1.1718981266021729, -1.54050874710083, -0.19881625473499298, 0.22795864939689636, -0.2658408284187317, -0.6767491698265076, 0.31082573533058167, -1.7119433879852295, -0.41921713948249817, -0.9713340401649475, -0.5420025587081909, -0.5434046387672424, 1.2813211679458618, 2.8781888484954834, 0.41858500242233276, -1.0929694175720215, -0.5606797933578491, -0.04313913732767105, -1.4719953536987305, -0.9014889597892761, 2.843994379043579, -1.016034722328186, 0.5591040849685669, 1.1049062013626099, 0.7702670693397522, -1.109093189239502, -0.5182532668113708, -0.08959459513425827, -0.762131929397583, 1.3136622905731201, 0.22608232498168945, -1.7375205755233765, 1.5797632932662964, 0.5224186778068542, -0.7296246886253357, -2.0390615463256836, -2.4123237133026123], [0.2909615635871887, 0.9338663816452026, 0.2390938550233841, -0.9075375199317932, -0.2463904619216919, -0.6664372682571411, 0.13168063759803772, 0.7827777862548828, -0.44860050082206726, 0.5328415036201477, 1.4900288581848145, -1.7684383392333984, -0.8380166292190552, -0.0015865728491917253, -1.1424777507781982, 0.12429829686880112, -0.2671244442462921, 0.43018171191215515, -1.1496646404266357, 1.177311658859253, 0.2690841853618622, -1.3438626527786255, 0.054473355412483215, 0.10268039256334305, 1.3994771242141724, 0.8117371201515198, 1.705755591392517, -0.24143408238887787, 0.15809567272663116, 0.35435813665390015, -1.1382851600646973, 0.6984377503395081, 0.41933220624923706, 0.13630647957324982, 2.0830271244049072, 0.8875256776809692, -0.17893949151039124, 1.6739709377288818, 0.17437146604061127, -0.09596692025661469, 0.4960784316062927, -0.09843531250953674, -0.8072085380554199, -0.9534953832626343, 0.6119527816772461, -0.8501517176628113, -1.528906226158142, 2.8140711784362793, 0.30050456523895264, -0.6354414224624634, 1.3382298946380615, 1.4721163511276245, -1.1629557609558105, -1.9199999570846558, 0.2533032298088074, 1.3082964420318604, -0.9491913914680481, 0.1750880628824234, 0.678078293800354, -0.5599623918533325, -1.0035361051559448, -0.5364800691604614, 1.63810133934021, -1.1047468185424805, 1.5324128866195679, -1.4823389053344727, 0.7101449370384216, -0.17634829878807068, 0.7408108711242676, 0.12945176661014557, -0.7174078226089478, -0.09532114863395691, 1.2384814023971558, 0.7304605841636658, 1.6223037242889404, 1.104284644126892, 1.8986194133758545, -0.6357982754707336, 1.2877987623214722, -0.7759032249450684, -0.5395125150680542, 1.6719247102737427, 0.024765346199274063, 1.2500369548797607, -0.24615225195884705, 0.4911757707595825, 1.337135910987854, -0.17770305275917053, 0.027274101972579956, -0.3767217695713043, -0.5898860096931458, 1.0760923624038696, 1.1202727556228638, 1.2587289810180664, 0.18223348259925842, -0.26487526297569275, -0.11209748685359955, -0.2875472605228424, 0.1860036849975586, -1.0683857202529907, -1.5728890895843506, -0.6787395477294922, -0.17955458164215088, -0.3747764229774475, 1.7225754261016846, 0.0788656547665596, 1.156704068183899, -0.9749073386192322, 2.0992727279663086, 0.3041302263736725, -0.8066221475601196, -0.013015266507863998, -1.80286705493927, -3.416686773300171, -0.3576158285140991, -0.10364347696304321, -0.9786799550056458, -0.9078721404075623, 0.8453399538993835, 1.4389582872390747, -1.739378809928894, -0.3175463080406189, -0.5646426677703857, 0.7196196913719177, -1.1565696001052856, 0.39798083901405334, 0.3188716471195221, 0.4853891134262085], [0.6431387066841125, -0.3707495927810669, 0.9335200190544128, -0.878934919834137, -0.6352190971374512, 1.0161511898040771, 0.3022862374782562, 0.4641667306423187, 0.38722050189971924, 0.41244062781333923, -0.7897436022758484, 0.8052492737770081, -0.5984559655189514, 0.170438751578331, -0.2635613679885864, 0.8888668417930603, -1.8996472358703613, -1.524544358253479, -1.3318474292755127, -0.31985363364219666, -0.6084485054016113, -0.7868567705154419, 0.33131906390190125, 0.8542018532752991, -1.0975590944290161, -0.998594343662262, 1.449332356452942, 0.7309174537658691, 0.9721726775169373, -1.0971654653549194, 1.3281745910644531, -0.8517775535583496, -0.7930541038513184, -1.27196204662323, 0.27945154905319214, 0.592043936252594, -1.184876561164856, 0.5614251494407654, 0.5766255855560303, 0.11156947165727615, 0.7443427443504333, 0.6568681597709656, 0.3693262040615082, 1.2447320222854614, 0.7151384353637695, 0.49645620584487915, -0.8861002922058105, -1.1501163244247437, -1.491695761680603, 0.7092275023460388, -0.5576045513153076, 0.08006314188241959, 0.1291281282901764, -0.9640865921974182, -2.2660579681396484, 0.14286981523036957, -0.2364169806241989, 0.9701638221740723, -0.07795284688472748, -0.042659275233745575, -1.0636638402938843, -0.7937313914299011, -0.4905537962913513, 0.12134578824043274, 0.5377913117408752, -0.1735636591911316, -0.8406689763069153, -0.19453229010105133, -1.0654067993164062, -0.11623343080282211, -0.27514412999153137, 0.4991486072540283, -0.6011613607406616, -0.2884069085121155, -0.12922486662864685, -0.5994147658348083, -0.9926010966300964, 0.6722073554992676, 1.250241994857788, -2.0675857067108154, -0.42831575870513916, 2.51216983795166, -2.1349265575408936, -0.9598783850669861, 0.5457908511161804, 3.3781166076660156, -0.365299254655838, -0.16256695985794067, -0.9805311560630798, -2.6357312202453613, -0.31788933277130127, -0.0692218691110611, -1.2346224784851074, 0.22185413539409637, -1.0742392539978027, 0.3688880205154419, -1.836182713508606, 0.07382599264383316, -1.006721019744873, 1.1527968645095825, 1.5454355478286743, -0.814437985420227, 0.27193912863731384, 0.023976150900125504, 0.6092526912689209, -1.3383270502090454, -0.09816331416368484, 1.8831337690353394, 0.6888591051101685, 0.58808833360672, 0.7229672074317932, -0.5522143840789795, 1.278527021408081, 0.6928725242614746, 0.6415296196937561, -1.3671389818191528, 0.4916543960571289, -1.604019045829773, -1.6909348964691162, -0.3103034794330597, 0.3506254255771637, 1.6188725233078003, 0.17736782133579254, -0.8636842370033264, -0.002189414110034704, -0.13742494583129883, 1.0199686288833618, -0.4493253827095032], [0.07940371334552765, 0.5501047372817993, 0.42915868759155273, 0.6563741564750671, 0.8967881202697754, -0.23925818502902985, -0.05936987325549126, 1.1200487613677979, 1.3108761310577393, 1.5300995111465454, 0.04857361689209938, 0.3427867889404297, 1.3225903511047363, -0.4450639486312866, 0.08467824757099152, -1.3351562023162842, -0.9772726893424988, -1.050162434577942, 0.28989818692207336, -1.606735110282898, 1.122772455215454, 0.2141803503036499, -0.0934562161564827, -1.1452150344848633, -1.37288236618042, -0.7872341871261597, 0.39525356888771057, -1.5035462379455566, -0.25414738059043884, 0.49643734097480774, 0.6097638607025146, 1.7699576616287231, -1.2686588764190674, -0.7018032073974609, 0.7780599594116211, -1.2349085807800293, 0.011402416974306107, -0.6731964349746704, 1.2653414011001587, -0.5732799768447876, -0.12320956587791443, 0.0747971311211586, 0.2501203417778015, -1.1668787002563477, 0.8579298853874207, 0.5928037166595459, 0.20793195068836212, 0.09449371695518494, 0.2208622395992279, 0.08342232555150986, -0.13150882720947266, 0.49127236008644104, -1.7341351509094238, 0.947941780090332, 0.3104705810546875, 0.16063600778579712, 0.4967553913593292, 0.4726274013519287, 1.8793240785598755, 0.1996065378189087, -2.0604045391082764, 0.3046261966228485, -0.5638979077339172, 0.13506947457790375, 2.0323851108551025, -1.0788335800170898, 0.8607605695724487, 0.12735968828201294, -2.684408187866211, 0.6353060603141785, 0.8835340738296509, 0.2839919328689575, 0.26314684748649597, -0.5871113538742065, -0.6804817914962769, 0.8678175806999207, -2.2178971767425537, 0.11300411820411682, -1.4452658891677856, -0.7141289114952087, 0.8760672807693481, 0.9506092071533203, -0.5589949488639832, -0.7604793310165405, -0.6824055910110474, 1.2380361557006836, -0.28826427459716797, 0.46623387932777405, 1.1209666728973389, -1.4919618368148804, 0.8030998110771179, 0.7162826061248779, 1.5892590284347534, 0.6022542715072632, -0.3041461110115051, 2.171844720840454, 1.312551498413086, -0.18560504913330078, 1.6469426155090332, -1.881137728691101, -0.9370231628417969, 0.07945969700813293, 0.20536968111991882, -0.466461181640625, 0.11957038938999176, 0.24402353167533875, 2.1759495735168457, 0.5515612959861755, 0.8984830379486084, -1.0621916055679321, -2.3886842727661133, -1.0977922677993774, 1.4131220579147339, 0.300306499004364, 1.0597050189971924, -1.6029260158538818, -1.2047796249389648, -0.29763564467430115, 0.22233782708644867, 0.5813698768615723, 0.6591854691505432, 0.1903172880411148, 0.830646812915802, 0.6454152464866638, -0.9078510403633118, -0.0011576170800253749, 0.7040822505950928, 0.347173810005188], [-0.030320681631565094, -1.7368446588516235, -1.5157238245010376, 0.6726182103157043, -0.9797017574310303, 0.3121863007545471, 1.3453670740127563, -0.14292091131210327, 1.075465440750122, -0.6883081793785095, 1.2661582231521606, -1.3636974096298218, 0.06393100321292877, -0.4976578652858734, -1.5946176052093506, 0.22158150374889374, -1.32462739944458, 0.7687506675720215, 1.2358192205429077, 0.187626451253891, -0.8107911348342896, 0.9334622025489807, 0.4809955954551697, -0.29323285818099976, -0.1648954302072525, -0.2816360890865326, -0.6798608303070068, -0.41993144154548645, 0.702294647693634, 0.204660564661026, -0.13143236935138702, 0.9794161319732666, -0.83054518699646, -0.1302415132522583, 1.558478832244873, 0.22827622294425964, -0.43975576758384705, -1.8651808500289917, 0.4046270251274109, -0.6635130643844604, -0.09016883373260498, -1.7845768928527832, 0.4962818920612335, 0.40025320649147034, -1.562178373336792, -0.5254811644554138, -1.0359641313552856, -0.49702829122543335, 0.179144486784935, 0.3687998950481415, -0.09693492203950882, -0.3733932673931122, -0.32818636298179626, 1.9551328420639038, -0.4132821261882782, -1.5814019441604614, 1.6625605821609497, 1.2629680633544922, -0.9255849123001099, 2.0249199867248535, -0.06840355694293976, -0.22013813257217407, -0.9543216228485107, -0.5300253629684448, 1.1985145807266235, -1.3754115104675293, -0.2688523828983307, -0.9380852580070496, -0.003997525665909052, -1.0639795064926147, -0.9856317043304443, -0.5925927758216858, -1.3124984502792358, 0.7158148884773254, -0.23627202212810516, 0.7003538012504578, -0.6732885837554932, 0.019145911559462547, -1.842034935951233, -0.3628681004047394, -0.4980182945728302, -0.18061463534832, 1.5322266817092896, 0.00653518782928586, -0.3405597507953644, -2.011824607849121, 0.49180033802986145, -0.9112981557846069, 1.8545745611190796, 0.013082697987556458, 1.3832483291625977, 1.5459120273590088, -0.07210691273212433, -0.935614824295044, 0.6514905095100403, 0.7444769144058228, 0.18981419503688812, -0.16530735790729523, 0.31480053067207336, 0.17063291370868683, -0.8813748359680176, -0.21988128125667572, 1.0164726972579956, -0.448254257440567, 0.4189058840274811, -0.08360730111598969, 0.22543838620185852, 1.5405353307724, 0.7396214008331299, 1.0943102836608887, -0.16310802102088928, 0.23194552958011627, 1.1764285564422607, 0.9359956383705139, 1.8113306760787964, 0.024776263162493706, 0.6424111127853394, -0.7458586096763611, 0.021207600831985474, -0.2140406370162964, 0.3189389407634735, 0.828833818435669, -1.057897925376892, 0.5831020474433899, 0.18767645955085754, -0.7877518534660339, 0.9809408187866211, -0.388657808303833], [0.2026934027671814, 0.6906387805938721, 1.9334323406219482, -0.6245592832565308, -0.765953004360199, 1.0237600803375244, 0.9855363965034485, -0.23075984418392181, -0.019038477912545204, -0.6019450426101685, -1.0411986112594604, -0.5872291326522827, 0.07743718475103378, 1.4904180765151978, 1.22101891040802, -0.25729086995124817, 1.0454678535461426, 1.5123963356018066, 0.6261261105537415, 0.34798359870910645, 1.088950276374817, 0.7134182453155518, 1.5533053874969482, -0.6224392652511597, 1.087160348892212, -0.9798054099082947, 0.56231689453125, 0.6477251648902893, 0.8880340456962585, -0.02316613867878914, 8.346988761331886e-05, 0.07990951091051102, -0.6173614859580994, -0.06549955904483795, -0.7789402604103088, -0.3634085953235626, 1.2224947214126587, 0.4318332374095917, -0.8950092196464539, 0.1503204107284546, 1.1475645303726196, -0.2637200951576233, -1.9443825483322144, -1.5923118591308594, -0.8401973843574524, 0.09468140453100204, 0.7661020755767822, -1.5182138681411743, -0.5225584506988525, 0.041127149015665054, 1.6964558362960815, 2.0274441242218018, 0.8587547540664673, 1.5903979539871216, 0.6315611004829407, 0.09298841655254364, -0.455181360244751, -0.8051801323890686, 0.06489715725183487, -0.15058469772338867, 0.25367897748947144, 1.2024815082550049, -0.819699764251709, 0.33183395862579346, 1.7852667570114136, 0.3243338167667389, 1.4803552627563477, -0.5564469695091248, -2.91978120803833, 0.48138776421546936, 0.22272545099258423, 1.1971107721328735, -0.7487099170684814, 0.2330574244260788, 0.06779198348522186, -0.549338161945343, -0.3166784346103668, -0.2565465569496155, -0.469649076461792, -0.7412768006324768, 0.08387783914804459, 0.14627531170845032, -0.6692228317260742, -2.1831929683685303, -0.21963736414909363, -0.9407050609588623, -0.7708550095558167, 0.4683423638343811, 1.0750819444656372, -0.18612100183963776, 1.2818291187286377, -0.9631600379943848, -3.1114540100097656, 1.1204105615615845, 1.1891180276870728, 0.3017059862613678, 0.38035500049591064, 1.6202069520950317, -1.5041717290878296, 0.377308189868927, 0.179209366440773, -0.5592906475067139, 1.6724610328674316, -1.1949752569198608, -1.5528230667114258, -0.7454426288604736, 0.28528082370758057, -0.5309140086174011, 0.4835749864578247, -0.7778880596160889, -0.5856271386146545, -0.2136060893535614, -0.9123784899711609, 1.292798638343811, -1.5123605728149414, 1.1323012113571167, 0.2209181785583496, 2.526655673980713, -0.7666530013084412, 1.2873705625534058, -1.2781362533569336, 0.1974315345287323, -1.9501290321350098, 1.5778818130493164, 0.974761962890625, -0.641940176486969, -0.5425825715065002, -0.43676674365997314], [0.31880319118499756, -0.34970149397850037, -0.3049179017543793, -0.4322573244571686, 0.748021125793457, -1.2162257432937622, 1.9042738676071167, 1.3384733200073242, 1.8511736392974854, 0.45549312233924866, 0.11799976974725723, -0.19235335290431976, -0.8695381879806519, -0.9988535046577454, 0.19235116243362427, -0.4204532504081726, 1.8601274490356445, 0.8653858304023743, -0.4703249931335449, 0.7978596091270447, 1.4643150568008423, 1.124879240989685, 1.125731348991394, 0.5862370133399963, 0.4903525114059448, 1.810503363609314, 0.7101044058799744, 1.7296241521835327, 0.9491735100746155, -1.0588698387145996, 1.0408210754394531, 1.6395336389541626, 0.16393204033374786, -1.8351504802703857, 1.4956053495407104, -0.5929845571517944, -1.5013257265090942, -1.5627986192703247, 0.12418168038129807, 0.5581939816474915, -0.923209011554718, 0.09469932317733765, 0.09945198893547058, 1.6981403827667236, 0.4744773209095001, 0.6891348958015442, 1.1688381433486938, -0.773389995098114, 0.24199353158473969, -1.740695595741272, 0.06322336196899414, 1.342268943786621, -2.070953607559204, -0.3221340775489807, 0.8770272731781006, 1.886448860168457, 0.9437713027000427, -0.6312077641487122, 0.8579921722412109, 1.2737596035003662, 0.7699694633483887, 1.522127389907837, 2.9073548316955566, -0.30179524421691895, 0.7085548639297485, -1.0440504550933838, 0.39742016792297363, -0.8785706758499146, -0.24817794561386108, -1.0560634136199951, 2.0476009845733643, 1.7458462715148926, 1.8356947898864746, 0.2349967062473297, 0.49840590357780457, -1.3232537508010864, 1.3675576448440552, -0.897579550743103, -0.45340868830680847, 0.5404161810874939, -0.3311145305633545, 0.4561440944671631, 0.6893104910850525, 0.5545542240142822, -0.6757873892784119, -0.6000633239746094, -0.8898217678070068, -2.548891305923462, -0.5333441495895386, -0.6128318905830383, 1.138569951057434, -0.03510013967752457, -0.8934636116027832, 0.26824647188186646, -1.4648218154907227, -0.4542985260486603, 1.1582988500595093, -1.0750333070755005, -0.5107864141464233, 0.20457352697849274, 0.05428025871515274, 1.473671555519104, 0.23392826318740845, -0.43647822737693787, 1.0228568315505981, -0.17647920548915863, 0.059658680111169815, 0.1644916981458664, 0.22554011642932892, -1.2702304124832153, 0.5656086802482605, 0.25723811984062195, 0.806877851486206, -1.738316535949707, 0.5476157069206238, -0.09103307127952576, -1.3357361555099487, -2.450364351272583, 0.35575568675994873, 1.3900463581085205, -0.9235349297523499, 0.5137325525283813, -0.43762198090553284, 0.7821213006973267, 0.5826863646507263, -1.277435302734375, 0.7406367063522339, 0.03354497253894806], [0.6670389771461487, 0.26689985394477844, 0.9683048725128174, 1.0854192972183228, -1.3681583404541016, -0.1698121428489685, -0.7620118260383606, 1.5774030685424805, -0.5847107172012329, -2.17147159576416, -0.7300528883934021, 0.27866560220718384, -1.562053918838501, 2.6724860668182373, 0.4048272967338562, -0.7138912081718445, 0.7278752326965332, -0.7118419408798218, 0.29632502794265747, -0.23418255150318146, -0.5457959175109863, -0.9969808459281921, -0.4037201404571533, 0.9790439009666443, -1.1533498764038086, 0.17839276790618896, -0.42005106806755066, -0.6349347233772278, 0.948842465877533, -0.4861529767513275, 0.3713986277580261, 0.42702892422676086, -1.13490891456604, 0.17808088660240173, 0.574589192867279, 0.4871894121170044, -1.2230169773101807, 0.37483832240104675, -3.4190967082977295, -2.2241411209106445, -0.6624370813369751, 0.27167338132858276, -1.5823696851730347, -1.7906029224395752, -0.8195711970329285, 0.4029610753059387, -0.16662082076072693, -0.03666062653064728, -0.16013844311237335, 0.4068696200847626, 0.011479564942419529, 1.1908968687057495, 1.1621443033218384, 0.09249398857355118, 1.2310206890106201, -0.16161994636058807, -1.1619701385498047, 0.49822187423706055, -0.9413446187973022, -0.06725894659757614, 0.5596003532409668, -0.7575998306274414, -0.3343774974346161, 0.5880276560783386, -0.3645079433917999, -1.9181627035140991, -0.7589508891105652, -2.3316075801849365, 0.8477463722229004, 0.13270312547683716, -0.6472005844116211, -0.738431453704834, -0.13648682832717896, 1.5713708400726318, 0.01172194629907608, 1.2654715776443481, -0.7898964881896973, -0.6821109056472778, -0.09995269775390625, 1.4856306314468384, 1.7231594324111938, 0.023226531222462654, 0.04526253044605255, -0.3615024983882904, -1.4287163019180298, -0.10086707770824432, -2.2153923511505127, 1.766038179397583, -0.1458812952041626, -1.481680154800415, 2.3346025943756104, 0.5360125303268433, 0.5311242341995239, -1.7344411611557007, -0.428772509098053, -0.91136234998703, 0.32772883772850037, 0.7943174242973328, -0.4266098439693451, 0.411933571100235, -0.6788775324821472, 0.14614185690879822, 0.08593904227018356, 0.37894392013549805, -0.7692762017250061, -0.08969101309776306, 0.03402415290474892, 0.8902423977851868, 0.2971270978450775, 0.9898414015769958, 0.8124328851699829, 1.7139968872070312, -1.9185798168182373, -0.2733319401741028, -0.08425384759902954, 0.4310459792613983, 0.4208334982395172, -0.9179670214653015, 0.13721896708011627, 2.5857176780700684, 1.0764909982681274, -1.358414888381958, 1.1344603300094604, 0.06942200660705566, -0.7080109119415283, -1.5244190692901611, -1.4275263547897339, 0.41105401515960693]], [[0.11738217622041702, -0.28452953696250916, -0.25643494725227356, 0.37069588899612427, 1.6335737705230713, -0.5187612771987915, 0.2988796830177307, -0.5340548753738403, -0.2008253037929535, 1.8774808645248413, -0.8917354941368103, 0.23413221538066864, -1.9321234226226807, 0.6452655792236328, 0.24333685636520386, -0.29961448907852173, -0.3674311637878418, -1.138014554977417, -0.1782420128583908, 1.1606149673461914, -1.2056217193603516, 1.0264537334442139, -1.3774532079696655, -1.8893826007843018, 0.14627686142921448, 1.0425493717193604, 1.576013445854187, -0.0832352340221405, 0.00246595311909914, 0.32766440510749817, 0.416951060295105, -1.2980382442474365, 1.4567298889160156, 0.3740122318267822, -0.9826667904853821, -1.9194321632385254, -1.6876769065856934, 0.4287025034427643, -0.04970988258719444, -0.02465488202869892, -0.037669677287340164, -0.28155240416526794, -1.4165699481964111, -1.0191274881362915, 0.017908789217472076, -1.0995454788208008, 0.7895089983940125, 0.6463949084281921, 0.07365212589502335, -0.29885855317115784, 0.7649855017662048, 0.856507420539856, -0.7094451189041138, -0.8241392374038696, -1.427182912826538, 0.007819618098437786, -0.2538403868675232, 0.6655129194259644, 0.34814754128456116, -0.9935615062713623, 0.3139974772930145, -1.795021891593933, 0.7140526175498962, 0.3392705023288727, -1.520817756652832, 0.06171228364109993, 1.2975547313690186, 1.310370683670044, 0.7829875349998474, -0.10614456981420517, 0.44795477390289307, -0.3788999021053314, 0.668707549571991, -0.044586360454559326, -1.2981454133987427, 0.20257531106472015, -1.366077184677124, 0.49873194098472595, 1.7038087844848633, -1.688527226448059, 2.0363657474517822, -0.7406644821166992, -1.0354053974151611, 0.6859853863716125, -0.08798400312662125, 0.6057551503181458, -1.2408843040466309, -1.7349132299423218, 1.0786969661712646, 0.030380740761756897, 0.2174028605222702, -0.9141533970832825, -0.37874436378479004, -1.6544179916381836, 0.8739286065101624, 0.22148160636425018, 0.5094283223152161, 0.39715293049812317, 2.1369869709014893, -0.7673510909080505, 0.37571442127227783, -0.23457267880439758, 1.0256106853485107, 0.9683157205581665, 0.03175411745905876, 0.29461240768432617, -0.4569430947303772, -0.8218176364898682, -0.15274739265441895, -0.08954036980867386, 0.19819721579551697, 0.5317929983139038, -0.8963575959205627, 1.5191985368728638, -0.6911364197731018, -0.733609676361084, -0.412244588136673, 0.516074538230896, 0.035421065986156464, 0.07744606584310532, 0.9254966378211975, -0.5411695837974548, -0.8171749711036682, 2.1944100856781006, 0.09606817364692688, -0.748985230922699, -0.5333720445632935, 0.5095811486244202], [0.5656719207763672, 1.6501976251602173, -0.1757407784461975, 0.4081610441207886, 0.7844440937042236, -0.6833357810974121, -0.7051281929016113, 0.722196638584137, 1.0200064182281494, 2.444640874862671, -2.0326952934265137, -0.7090276479721069, -0.8083912134170532, 1.1251733303070068, 0.8760108947753906, -0.2177010327577591, -1.9365495443344116, -1.302026391029358, 1.2539936304092407, 1.4385428428649902, -0.4304104149341583, 0.545994758605957, 0.05124916508793831, 0.2741544246673584, 0.7591817378997803, -0.5266432762145996, 1.3850104808807373, -0.06807132810354233, -0.7562919855117798, 0.21756881475448608, -0.12577655911445618, 1.8663586378097534, 1.690227746963501, -0.7086697220802307, 2.281543016433716, 2.0854744911193848, -0.48010188341140747, 1.7939722537994385, 0.2925475537776947, -0.753116250038147, 1.0300017595291138, 1.2481446266174316, 0.5335443019866943, -0.4187106490135193, -0.10110097378492355, 0.2863514721393585, -1.0750643014907837, -0.346331387758255, -0.14480489492416382, -1.5974231958389282, -0.11814795434474945, 2.126037120819092, 0.01377064362168312, 0.004799245856702328, -0.13168717920780182, -0.5003218054771423, 1.0701465606689453, -1.179400086402893, 0.19735516607761383, 0.0815582424402237, -1.5768582820892334, 1.0385534763336182, 0.01827976480126381, 0.16614699363708496, -1.0330089330673218, -1.1730601787567139, 0.7753342390060425, -0.016531523317098618, -0.48754018545150757, 0.23441003262996674, -0.027316056191921234, -0.058038145303726196, 0.10607326775789261, -0.7945929169654846, 1.4337106943130493, -0.9921513199806213, -1.4046343564987183, 0.10562087595462799, -0.4520806670188904, 0.4414496123790741, -0.7160648703575134, 0.8456874489784241, -0.8496333956718445, 1.1396374702453613, -0.559646725654602, -0.7667518854141235, -1.3308167457580566, -2.038550853729248, -0.140292227268219, -2.9629483222961426, 0.757599413394928, -0.9254327416419983, -2.4456822872161865, 1.0161019563674927, -0.8277366757392883, -0.9545183777809143, -1.010021448135376, -0.23986537754535675, -0.31672006845474243, -1.2015496492385864, -0.207622230052948, 0.6612286567687988, 1.3401298522949219, 1.358294129371643, 0.9107419848442078, -0.032967645674943924, -0.5812108516693115, 1.0805182456970215, 0.33343103528022766, 0.8406151533126831, 0.49143460392951965, 1.504483699798584, 0.9855679273605347, 1.0109671354293823, -0.22520127892494202, 0.2881968915462494, -0.21273182332515717, 0.16332510113716125, -1.8889254331588745, 0.3814254403114319, -1.7148383855819702, -1.667173147201538, -0.14219728112220764, -2.237032413482666, -0.9408167004585266, 1.416420817375183, 0.2737780809402466, 0.9731608629226685], [-0.12227120995521545, 0.10247616469860077, 1.1551049947738647, -2.8088579177856445, -0.8776087760925293, 0.20628361403942108, -0.6073667407035828, -0.16705437004566193, 1.4473975896835327, -0.8009686470031738, -1.9895704984664917, -0.9848519563674927, 1.6978017091751099, 0.21544940769672394, 0.3173368573188782, 0.48013824224472046, -1.5940091609954834, 0.2115171104669571, 0.4807562828063965, -1.9724061489105225, 1.7572578191757202, -0.3377498388290405, 0.14237312972545624, -0.2682439684867859, -0.8179821968078613, 1.3060811758041382, -0.8793309926986694, -0.4655747413635254, 1.9623388051986694, 0.11024352163076401, -1.684876561164856, -0.35553476214408875, -0.5310198068618774, 1.601638674736023, -1.57395601272583, 0.08040733635425568, 1.5547094345092773, 0.2568247318267822, 0.003082594834268093, -0.8950788378715515, -0.583434522151947, -0.5251625180244446, 0.38106897473335266, -0.1994408518075943, 1.207121729850769, 0.034382760524749756, -0.08097195625305176, -0.8541483879089355, -0.8828360438346863, 1.0317951440811157, 0.7623845934867859, -1.376513123512268, -0.10308366268873215, 1.0692729949951172, -0.5872623920440674, 0.749726414680481, 0.9992414116859436, -0.22358787059783936, -0.5121302604675293, 0.6003074049949646, -0.8390719294548035, -0.34433451294898987, -0.2175855189561844, -0.9009233713150024, -0.020737409591674805, -1.190327525138855, 0.27914413809776306, -0.2887115180492401, -0.8680531978607178, -0.5509130954742432, 0.5358132719993591, -0.971327006816864, -0.37290510535240173, -0.1918969452381134, -1.2457034587860107, -0.6499612927436829, 1.2865480184555054, -0.0037632633466273546, 0.731990396976471, 2.335529088973999, 0.7072285413742065, 0.6504451036453247, -1.3369024991989136, 0.10352376103401184, -1.2619599103927612, -0.16310174763202667, -2.824918508529663, 0.09462273120880127, 0.18713189661502838, 1.1391551494598389, -0.011449221521615982, 0.055342938750982285, -0.1542913019657135, -1.0624200105667114, 0.012394889257848263, -1.634657382965088, -1.0792722702026367, -0.6374289393424988, 0.6002832651138306, -0.3210045099258423, -1.8335798978805542, -0.07153313606977463, 0.6148070693016052, -0.42840415239334106, -0.07443629950284958, 0.03726877644658089, 0.015100711956620216, -1.3904739618301392, -0.030019745230674744, 0.4096629321575165, 0.2704857885837555, 1.235691785812378, -0.20339521765708923, -1.1310237646102905, -1.394579291343689, 1.6108320951461792, 0.18499256670475006, 0.6385555267333984, 0.472059041261673, -0.06601817905902863, 0.053503263741731644, -0.043941181153059006, -0.050608936697244644, 1.457341194152832, -0.08503947407007217, 0.3596336841583252, 0.05303594097495079, 0.7733772397041321], [0.08488783985376358, -1.671923279762268, -1.264031171798706, 0.3888055384159088, 0.21397463977336884, 1.6197192668914795, -0.6311836242675781, -1.891005039215088, 0.06593940407037735, 0.46814045310020447, 1.3403347730636597, -0.5579165816307068, 0.05621535703539848, 0.340409517288208, -2.416616201400757, 1.8830434083938599, -0.05162004008889198, 0.1912783980369568, 1.5541001558303833, 0.8374976515769958, 1.1574373245239258, 1.0884618759155273, 1.3715919256210327, -1.0908658504486084, 1.0153683423995972, 1.3743765354156494, 0.3783911466598511, 0.6539704203605652, 0.532383382320404, 0.5771792531013489, -0.3925626277923584, -1.1566377878189087, -0.31759700179100037, -1.9035512208938599, 1.8116618394851685, 0.5146845579147339, 1.452649474143982, -0.6029017567634583, 1.3806872367858887, -0.14835669100284576, -1.013739824295044, 1.7862643003463745, 0.0627698004245758, 1.9120862483978271, 0.3362976014614105, -1.2712122201919556, -0.23498667776584625, 0.10145340114831924, 1.0471879243850708, -0.31644728779792786, 0.28937438130378723, 0.9189870953559875, 2.154754877090454, -0.05869916453957558, 0.3019336462020874, -0.8014488220214844, 0.3496197760105133, -0.22424441576004028, -2.2824039459228516, -0.06318660080432892, -2.4534268379211426, -1.152297854423523, -0.09155379235744476, 1.4107099771499634, 0.5386040806770325, 1.5208526849746704, -0.06072114035487175, -0.9797869324684143, -0.3929639160633087, -1.5228320360183716, 1.3205398321151733, 0.7402842044830322, 1.941508173942566, 0.2288946658372879, -0.037915993481874466, -0.6740268468856812, -0.665144681930542, 1.1990565061569214, -0.14079806208610535, 1.8238108158111572, 0.4927183985710144, -0.7510911822319031, 1.0525474548339844, -1.71793532371521, 1.2948657274246216, -0.5900337100028992, 0.8410466909408569, -0.7292947769165039, 0.17721576988697052, 0.1447737216949463, 0.6565196514129639, -0.11038349568843842, 1.690353512763977, 0.868668258190155, 0.38317927718162537, 0.055595315992832184, 1.0225389003753662, -1.59038507938385, 0.7363277077674866, -2.354454755783081, 1.4521771669387817, -0.3446812033653259, -0.20261165499687195, 0.17157818377017975, -0.07639133930206299, 0.9802597165107727, -0.630741536617279, -0.2525309920310974, -1.0146806240081787, -0.034031495451927185, 1.240903615951538, -1.2900069952011108, -0.5755882263183594, -1.4500327110290527, 0.8458544015884399, 1.7137974500656128, 1.487034797668457, 0.3911077380180359, 2.4433062076568604, 0.2049500197172165, -1.9923368692398071, 1.4797908067703247, 0.7531202435493469, -0.4817923903465271, 0.7266111969947815, -1.655877709388733, -0.707034170627594, -1.006754994392395], [1.6728311777114868, 1.4485539197921753, -2.5585720539093018, -0.8943719267845154, -1.3323370218276978, -1.1393176317214966, -2.1548874378204346, 1.5198595523834229, 0.8894383907318115, -0.583309531211853, 0.27464962005615234, 0.1276487559080124, 0.911742627620697, -0.4038734436035156, 0.3619498610496521, 1.79379403591156, 1.6069973707199097, -0.7759314179420471, -0.4024173617362976, -2.094141721725464, -1.171393871307373, -0.4171517491340637, 0.9540921449661255, 0.8796599507331848, 1.5211228132247925, 0.5532175302505493, 2.3956894874572754, 1.5369259119033813, -0.630593478679657, 1.2214213609695435, -0.9271698594093323, -0.1411532610654831, -0.7574848532676697, -0.313067227602005, -0.9467447996139526, -0.46485579013824463, 0.14728175103664398, -0.5615279674530029, -0.5966064929962158, 0.4574776589870453, 0.20883171260356903, 0.7290701866149902, -0.4624782204627991, -1.6392316818237305, -1.1811848878860474, 0.563589870929718, 1.1321749687194824, 1.3541163206100464, 0.9447062611579895, 0.40549641847610474, -0.5893660187721252, 0.6243466138839722, -0.8748548626899719, -1.1579279899597168, 0.10878255218267441, 1.7618111371994019, -1.110369324684143, -0.1366344839334488, 0.8745839595794678, 0.5735336542129517, -0.30615338683128357, 1.150228500366211, -0.9375376105308533, -2.002946138381958, -0.45123574137687683, -0.7066085338592529, -1.0225235223770142, -1.081792950630188, 0.7576206922531128, 0.5648878216743469, -0.6309213042259216, 1.5253583192825317, -1.0644842386245728, -1.8649202585220337, -0.13375456631183624, -0.8315325379371643, 1.337327480316162, -0.35677632689476013, -0.9712457060813904, -1.3181755542755127, -0.18501193821430206, -1.1614127159118652, 0.6204254031181335, 1.4749462604522705, -0.35963770747184753, 0.185737743973732, -1.4745603799819946, -1.2125736474990845, -1.8428144454956055, -1.3603167533874512, 0.7898721098899841, 0.32967865467071533, -1.5188144445419312, -1.0280689001083374, -0.2354988157749176, -1.1791043281555176, -0.4556503891944885, 0.9308029413223267, 1.6570178270339966, 0.562389075756073, 0.1929066777229309, -0.8004661202430725, -0.2879619598388672, -0.24900591373443604, -0.21278312802314758, 0.43085047602653503, 1.096479892730713, -0.03846515342593193, 1.20195472240448, -1.0601563453674316, -1.3446825742721558, 0.6895497441291809, -1.3069056272506714, 0.707452118396759, 1.3067742586135864, -0.7551349997520447, -1.0457160472869873, -1.3542869091033936, 0.05328156426548958, -1.0002621412277222, -0.11236759275197983, -0.6243605613708496, 1.074263334274292, -0.636076807975769, 1.7093682289123535, 0.19215108454227448, -0.5341366529464722, 1.623855710029602], [-0.5389488339424133, 1.084226369857788, -0.22095736861228943, 0.17555032670497894, -0.34093326330184937, 0.4755953252315521, 0.7758156061172485, 0.4350847005844116, -1.4473354816436768, 1.7822911739349365, 0.09011869877576828, -1.208999514579773, 1.871212363243103, 0.4005013406276703, -0.12765154242515564, -0.0677853375673294, -1.6286572217941284, -1.0161454677581787, 0.5203171372413635, 0.5684651732444763, 0.5144355893135071, 0.5144380331039429, -0.570722758769989, -1.13953697681427, -0.45027846097946167, -0.8816969394683838, -1.9530694484710693, 0.0743374228477478, -1.6052401065826416, 0.03288300707936287, 0.978644073009491, -1.2490215301513672, -0.7704952955245972, -0.6093847751617432, 0.03649487346410751, -0.6688988208770752, 0.14377403259277344, -0.26684075593948364, 2.179309606552124, 0.2452721744775772, -0.32271409034729004, 0.8304451107978821, -0.37053799629211426, -0.5734895467758179, -0.9820836782455444, -0.8156343102455139, 0.12086109817028046, 1.3981884717941284, -2.1581714153289795, 1.3459010124206543, -0.23960046470165253, 0.13219515979290009, 1.52415931224823, -0.8665053844451904, -0.08127229660749435, -0.22546924650669098, 1.2882118225097656, -0.08281490951776505, 1.1054962873458862, -0.4640292823314667, -0.1569146066904068, -0.1027851477265358, -2.1457653045654297, 0.5597124099731445, 0.5291639566421509, 0.3244161605834961, -1.9434705972671509, 0.28115957975387573, -0.17187434434890747, 1.5709025859832764, 0.9686650037765503, 0.5530759692192078, -0.1292959600687027, 1.506909728050232, -0.9729659557342529, 1.7072237730026245, -1.5065714120864868, 3.110316753387451, -0.9176006317138672, 0.872263491153717, 0.6756049394607544, -1.517109751701355, -0.3503677248954773, -1.8064802885055542, -0.8876100778579712, -0.5821583867073059, 1.1197766065597534, 1.077872395515442, 0.9930815696716309, -0.3294685184955597, -0.6860388517379761, -0.7584745287895203, -1.3617651462554932, 0.8160693645477295, 0.7210496068000793, -0.6761510968208313, 1.2420729398727417, -0.3064934313297272, -0.6144939661026001, 0.16591022908687592, -0.6331456899642944, 1.018009066581726, -0.8358727693557739, 0.9096275568008423, -0.10009367018938065, 0.3344021439552307, 0.058034852147102356, 0.38098645210266113, 0.1000916063785553, 1.2189940214157104, 0.06935745477676392, -1.0152199268341064, 1.4963889122009277, 1.307927131652832, -1.2048548460006714, -0.2287120223045349, 1.4149049520492554, -0.2205180525779724, 1.4790935516357422, -0.8065246343612671, -0.8209725618362427, 0.0935782864689827, 0.1826271265745163, -0.48125937581062317, 1.2687592506408691, -0.2292400449514389, -0.42808303236961365, -0.990339457988739], [0.13423970341682434, 0.9375690221786499, 0.5742986798286438, -1.4145798683166504, -0.11733872443437576, 0.513627290725708, 2.0026278495788574, 1.3380632400512695, 1.7894682884216309, -1.8227168321609497, 0.26582446694374084, -0.362657755613327, -2.157109260559082, -0.1192602664232254, 0.08711054921150208, 1.2727431058883667, 0.6732532978057861, -0.18535414338111877, -1.0142070055007935, -0.4806787073612213, 0.7118718028068542, -0.8353721499443054, 0.17703087627887726, -2.2894914150238037, 0.2897183895111084, 1.869491457939148, 0.13812631368637085, -1.536918044090271, 0.46912139654159546, 0.3796493113040924, 1.0485544204711914, 1.9527448415756226, -1.311347246170044, -2.0430760383605957, 0.9485774636268616, -0.013228017836809158, -0.22973844408988953, 0.7768288254737854, -0.027193473652005196, 0.9622337818145752, 0.9626157879829407, -0.9544321894645691, -1.1042134761810303, -0.23700769245624542, 1.6021075248718262, 0.13208164274692535, -1.067550539970398, -1.6129813194274902, -1.1907505989074707, -0.002572101540863514, -0.43924233317375183, -0.02397337183356285, 0.12630559504032135, -0.1354232132434845, 0.5361692905426025, -0.40819820761680603, 0.2703537344932556, -0.16253389418125153, 0.21615993976593018, 0.9540842175483704, -0.36728736758232117, -1.6483311653137207, -0.12704089283943176, -0.18862873315811157, -0.0015564421191811562, 0.7944839000701904, 1.5501521825790405, 1.681523084640503, -0.7365191578865051, -0.7871273159980774, 0.17932680249214172, 1.2932389974594116, 1.1282870769500732, -0.19779878854751587, 0.35862743854522705, -0.9442029595375061, -0.3807515799999237, -2.1046950817108154, 0.8093056082725525, 0.8036876320838928, -0.5208267569541931, -0.428585410118103, 0.8727707266807556, -1.22637939453125, -0.17267674207687378, 0.0012426787288859487, -0.3587264120578766, -0.7316232919692993, 1.6983805894851685, -0.06437553465366364, -1.323528528213501, 2.4153947830200195, -0.21264363825321198, -1.6925718784332275, 0.5102208852767944, -0.5550726056098938, 1.7813369035720825, -0.8588209748268127, -2.6363606452941895, -0.7617028951644897, 1.0131453275680542, -1.2609848976135254, -0.1583770364522934, 1.181359887123108, -0.6719173192977905, -1.0370150804519653, -0.4001528024673462, 2.2153546810150146, 0.736062228679657, -1.3003901243209839, -1.2110508680343628, 0.15759223699569702, 1.147420883178711, -1.302887201309204, 0.3324405550956726, -1.0647326707839966, 0.2699233293533325, 0.6691559553146362, -0.497639000415802, -0.001792146940715611, -0.5600998997688293, -2.6622819900512695, -0.18572813272476196, -0.5134543776512146, -0.3565114140510559, -0.44580093026161194, -1.09366774559021, -0.5679590106010437], [0.18803349137306213, -0.9893667697906494, 2.2394373416900635, -0.4622330665588379, 1.3018237352371216, 0.7223829030990601, 0.018942462280392647, -0.18831001222133636, -0.8246868848800659, -1.0795999765396118, -0.645480751991272, 0.6068132519721985, 0.8430998921394348, 0.596074104309082, -0.3521166741847992, 1.174879789352417, 0.4009401798248291, 0.7798557281494141, 0.20918555557727814, 1.2526023387908936, 0.5471222996711731, 1.9558677673339844, 1.2031104564666748, 0.9753023386001587, -0.6782723069190979, -1.1542778015136719, 0.43057510256767273, -0.9959401488304138, 0.9135249853134155, 1.6922844648361206, 0.6460492014884949, 0.3493247628211975, 0.9951351284980774, 0.06549303233623505, -0.1989934891462326, 0.08138392865657806, -0.18592464923858643, -1.6689634323120117, -1.2669346332550049, -0.896876335144043, 1.0808024406433105, 0.1560182422399521, -0.18202117085456848, 0.5967822074890137, 0.323372483253479, 0.883151113986969, -0.1332607865333557, -1.9086474180221558, 1.0505019426345825, -1.2298989295959473, -0.4644121527671814, 0.43619465827941895, -0.01571039855480194, -0.14769355952739716, -0.4023490846157074, -0.49532556533813477, -0.12638939917087555, 0.7956096529960632, -1.3326654434204102, -0.3335489332675934, -0.34465351700782776, 0.932630181312561, 0.6833726763725281, 0.40191325545310974, 0.5318313837051392, -0.8237621188163757, -1.396794080734253, 0.40747055411338806, -1.1356279850006104, 1.0349339246749878, 0.76573646068573, 1.3267285823822021, -1.1747009754180908, -0.7662330269813538, 2.1803860664367676, 0.6372933387756348, 1.159605622291565, -0.12547393143177032, 1.9230859279632568, 0.18985608220100403, -0.5221319198608398, 0.25871965289115906, -0.20963898301124573, -0.9667688012123108, 0.6375941038131714, 2.5943806171417236, 2.323171615600586, 0.02463284507393837, -0.866558849811554, 1.2082910537719727, 0.2724888026714325, -0.6042906641960144, -0.6392096877098083, -0.6007975935935974, -0.23773297667503357, 0.6801725625991821, 0.38149744272232056, -0.5012771487236023, -0.5531905293464661, 0.8152554035186768, -0.16799229383468628, 0.43107321858406067, -0.8002694249153137, 0.05916250869631767, 0.35902711749076843, 0.10928601026535034, -0.8583585023880005, -1.208380103111267, 0.4336095452308655, 0.4817436635494232, 2.4658560752868652, -0.6653177738189697, 1.0632132291793823, 0.26574862003326416, 1.3683170080184937, -1.1748994588851929, -1.6272720098495483, -1.6340123414993286, -1.9418003559112549, 1.1482698917388916, 0.3726647198200226, -0.8816478252410889, 0.8785611391067505, 1.4451446533203125, -0.8947716355323792, -0.7519158720970154, 0.12176033109426498, -0.6500492095947266], [-0.24807369709014893, 0.1963277906179428, -1.3804823160171509, -1.0520933866500854, -0.5241790413856506, -1.5840294361114502, 0.6613433957099915, 1.883054256439209, -0.2776649594306946, 0.8620990514755249, -0.0047975946217775345, 0.6358075737953186, -1.2629767656326294, 1.2155548334121704, 1.8029729127883911, 0.6792271733283997, 0.07774252444505692, -0.34108594059944153, -0.9384840726852417, 0.07402929663658142, -1.2682775259017944, -0.31119996309280396, 0.8438705801963806, -0.974706768989563, -0.5963652729988098, 1.9597615003585815, -0.15161184966564178, 0.7014375329017639, 1.3567395210266113, 0.11190146207809448, -0.8105193972587585, -0.41919511556625366, -1.1422525644302368, 0.45181804895401, -0.8539207577705383, 0.22902046144008636, -0.5980926752090454, -0.04599585384130478, 0.6405586004257202, 0.681415319442749, 1.4829308986663818, 0.8289811611175537, -2.1237223148345947, 0.35775884985923767, 0.8784214854240417, -1.3721818923950195, -0.0019557105842977762, 0.16400350630283356, -0.6762923002243042, 1.1182719469070435, 0.6321773529052734, -0.08733757585287094, 1.2486480474472046, 0.5478620529174805, -0.2502143979072571, 1.0275499820709229, 1.7828025817871094, -0.3986884355545044, 0.17685841023921967, 1.658418893814087, -0.10271111130714417, -0.036933064460754395, -1.540755033493042, -1.485917091369629, 0.41624313592910767, 1.6198936700820923, 0.16582736372947693, -0.03879847377538681, -1.697489857673645, 0.6724876165390015, 0.6289288401603699, -1.9019590616226196, -0.46088460087776184, 1.461295485496521, -0.7664908170700073, -1.28816819190979, 1.5445009469985962, 0.1186429113149643, -0.4919583201408386, -1.5491039752960205, -0.4446871280670166, 0.25283220410346985, 0.5314562916755676, -0.24202102422714233, 1.5251108407974243, 0.7557932138442993, 1.3806204795837402, 0.031309355050325394, 0.1951112300157547, 0.15719106793403625, 1.8133199214935303, -0.43129459023475647, 0.384440541267395, 1.0654264688491821, 0.6637437343597412, -0.27608522772789, -0.7870912551879883, 1.862007737159729, 1.5465424060821533, 0.06785798072814941, 1.6749171018600464, 0.5461854934692383, -1.1082788705825806, -0.961549699306488, 0.45461249351501465, -0.7946377992630005, -0.5556106567382812, 0.7776158452033997, -0.7681475281715393, -1.8683274984359741, -0.28423938155174255, 1.6057429313659668, -0.12474536150693893, -0.9645246267318726, -0.20554983615875244, 1.8669984340667725, 0.3697311580181122, -0.9841843247413635, -0.8555154204368591, 0.9285097718238831, 0.666160523891449, 0.08363570272922516, -0.8048239946365356, -0.5134908556938171, 0.6847864389419556, 0.9998024106025696, -1.049951195716858, -0.7598459720611572], [0.6656841039657593, 0.3694629967212677, 1.3181675672531128, -0.0030020023696124554, 0.053227927535772324, 0.1590927243232727, 1.1543198823928833, -0.6171562671661377, 0.5115481019020081, -0.39673691987991333, 0.0030374382622539997, 0.23306696116924286, 0.908928394317627, -3.1562271118164062, -0.12765707075595856, 0.1124863252043724, 0.18585900962352753, 0.934468686580658, 0.44398993253707886, 0.33664172887802124, 0.8451126217842102, 1.2774831056594849, 0.3402882516384125, 0.3628788888454437, 2.0520734786987305, 0.8189202547073364, -1.1863313913345337, -1.9641571044921875, 0.48795732855796814, 1.085415244102478, -0.6443340182304382, -0.2550743818283081, -1.7899631261825562, -1.4778610467910767, -0.599734365940094, 0.4161206781864166, 0.04219089820981026, -1.4081268310546875, -0.124866783618927, -0.8820369839668274, 0.3569140136241913, -1.5092871189117432, -0.19679751992225647, 2.321418285369873, -1.6361836194992065, -0.19900797307491302, -0.9409142732620239, -1.3847935199737549, 2.427957773208618, -0.2928242087364197, -1.0797849893569946, -0.2691619396209717, 1.220171570777893, 1.287548303604126, 1.509937047958374, -1.0092889070510864, -0.1791263222694397, 0.5674762725830078, 2.0761566162109375, -0.4698503315448761, -1.6270867586135864, -2.1292431354522705, -0.19231460988521576, -0.23454764485359192, -0.6347657442092896, 0.03413529694080353, 0.24182547628879547, 0.09218760579824448, 0.1665833443403244, 0.5038490891456604, -1.2932254076004028, -0.8451567888259888, -0.3625837564468384, -0.4208063781261444, -0.8378294110298157, -0.30975502729415894, -0.14759542047977448, -0.3182341754436493, 0.02921193093061447, 2.5093283653259277, -0.8391176462173462, 1.4642713069915771, 1.171842098236084, -1.5063371658325195, -0.3261086344718933, 0.044165223836898804, 1.8838547468185425, 1.3376604318618774, 1.3806010484695435, 1.1925413608551025, 0.22084198892116547, -1.3843841552734375, -1.1115201711654663, -0.9277581572532654, -0.41418883204460144, 0.8578044176101685, 0.24302463233470917, -0.9163643717765808, 0.03269452974200249, 0.6277370452880859, 0.1641223281621933, 0.34353986382484436, 0.3554048240184784, -0.33859142661094666, -0.4442306160926819, -0.7330716252326965, 0.21557921171188354, 0.5512478351593018, 1.7569862604141235, -1.1505813598632812, 0.24572254717350006, -0.34057244658470154, -0.08111110329627991, -1.2242234945297241, 0.4322871267795563, -0.1443226933479309, -1.7088005542755127, -0.13844196498394012, -0.38550499081611633, 1.0354703664779663, 0.7456487417221069, 1.720258116722107, -0.01885523460805416, -0.34857386350631714, -1.4791439771652222, -0.6139125227928162, 0.907231867313385, -0.05825228616595268], [-0.14657585322856903, 1.0853736400604248, 1.740372657775879, -0.8219195604324341, 0.9483528733253479, 2.560453414916992, -0.6200816035270691, 1.1223526000976562, -0.570135235786438, 1.0791465044021606, 1.8377299308776855, 0.8952584266662598, 0.7835422158241272, -0.9017463326454163, 0.49934422969818115, 0.11503180116415024, -0.7400987148284912, 1.1791620254516602, 1.18584144115448, -1.091480016708374, -0.6967650055885315, -0.49972549080848694, 1.6881344318389893, -0.8357570171356201, 0.3048386871814728, 0.32750698924064636, 0.01921425200998783, 0.1982017457485199, -0.8116627931594849, -0.6186830997467041, -0.7904495596885681, -1.0170129537582397, 1.4461092948913574, 0.6478881239891052, 0.541530191898346, 0.4187523424625397, -0.2039278745651245, 1.8134727478027344, -0.3350788950920105, 0.9993944764137268, -0.3959903120994568, 1.0233709812164307, -0.5411818623542786, -0.4565166234970093, 1.0387042760849, 0.5698856115341187, 0.7618258595466614, 1.5474671125411987, 0.10314352065324783, -0.42381197214126587, 0.10263726860284805, 0.999315083026886, 0.16623161733150482, -0.7910593748092651, -0.03650370612740517, 0.09205107390880585, -0.07686436921358109, 0.040860865265131, -0.2249218374490738, 0.4084099531173706, 0.1277204155921936, 0.24917849898338318, 0.5288679599761963, -0.6321818828582764, -0.5974975824356079, -1.0718294382095337, -0.9349544644355774, 0.13923709094524384, -1.3709149360656738, -0.5179818272590637, 0.4067591726779938, 1.6066714525222778, 0.4899900555610657, -0.6457725763320923, 1.3291685581207275, -0.30785053968429565, -0.9155336022377014, 0.985560953617096, -0.5413616895675659, -0.02397254854440689, 0.8626099228858948, 0.908663809299469, -0.7990179657936096, -0.046184953302145004, 0.1497430056333542, -0.3392617106437683, 1.438134789466858, -1.1818673610687256, -1.0644283294677734, 0.3304813504219055, 1.6946262121200562, -0.5176008343696594, -1.1719340085983276, 1.11465585231781, -0.05443737283349037, -1.364923119544983, 1.4470494985580444, 0.5310398936271667, 0.37122905254364014, 1.137391448020935, 1.2976537942886353, -0.41608116030693054, -0.7145207524299622, -2.208231210708618, -2.05063533782959, 0.21745459735393524, 1.0674923658370972, -1.4641462564468384, -0.2875266671180725, -2.2480742931365967, 0.26368147134780884, 1.108124852180481, -2.0224602222442627, 0.1326245218515396, 1.9034816026687622, -0.11341710388660431, -0.6381062865257263, -0.1519278734922409, -0.34681791067123413, -1.1013365983963013, 1.9972212314605713, -0.8434483408927917, -1.7862423658370972, -0.8311233520507812, -0.8324559926986694, -0.8632334470748901, 0.31192371249198914, 1.2381763458251953], [0.46168383955955505, -0.30537188053131104, -0.2754516899585724, -1.224083662033081, -1.9890539646148682, -0.5726816654205322, -0.9965290427207947, 0.39753788709640503, 0.10110440105199814, 1.0294184684753418, -1.3157851696014404, 1.7964717149734497, -0.13103362917900085, -0.1837068349123001, -1.4867911338806152, 1.783165454864502, 0.16251921653747559, 0.19953247904777527, -0.1156148761510849, 0.6272085905075073, 0.8698862195014954, -1.435655951499939, -2.010789155960083, 0.03307357802987099, -0.9915995001792908, 1.7794039249420166, 0.48784253001213074, -0.13488256931304932, 0.7265862226486206, 0.9411681294441223, 0.004060063511133194, 1.3989776372909546, 0.009084327146410942, -0.056334201246500015, -0.6017005443572998, 0.038076672703027725, 0.7697473168373108, -0.0872168317437172, -0.2709527611732483, -2.0055296421051025, 0.18883250653743744, -1.1298984289169312, 0.8920020461082458, 1.3919329643249512, 0.048595067113637924, 0.2309088110923767, 0.11525668948888779, -0.1879207342863083, 1.0632528066635132, 0.4122997224330902, 0.8501003980636597, 0.924876868724823, 1.0407192707061768, -3.142666816711426, -0.5162444114685059, -0.12864291667938232, 0.7980791330337524, 0.6382358074188232, -1.170776128768921, -0.9917412996292114, -0.4116307199001312, 0.14336980879306793, 0.39102616906166077, 0.1772959679365158, 0.9159114360809326, -0.8903151154518127, -1.0796746015548706, -0.6160085797309875, -0.7205215692520142, -0.17468507587909698, 0.2980307638645172, -1.3375869989395142, -1.1540772914886475, 0.728628396987915, 0.794611394405365, -0.5621936917304993, -0.9882407784461975, 0.534228503704071, -0.5202963352203369, -0.011771297082304955, 0.12000752985477448, -0.9438003301620483, -1.014075517654419, -0.4878115653991699, 0.5253517627716064, 0.9743061065673828, -0.4609658420085907, 2.498584032058716, 0.41705021262168884, -1.1283737421035767, 0.6155629754066467, -1.5193818807601929, 0.5165935158729553, -0.9670253992080688, 0.562608540058136, -0.5208752155303955, 0.5875436067581177, -0.7951277494430542, 1.387553334236145, 0.893072247505188, -0.01990421861410141, -0.1699533611536026, 0.4688454866409302, -0.3061636984348297, 2.04117751121521, -0.11754161864519119, -0.16321370005607605, 1.0240414142608643, 1.4953211545944214, -1.94973886013031, -0.2717195153236389, -2.0571603775024414, -0.270232230424881, 0.11800985038280487, -0.5964910387992859, -1.1893516778945923, -0.508133053779602, 0.7958914041519165, 1.2568109035491943, -0.7959411144256592, -0.06935907900333405, 0.7367619872093201, 0.113163061439991, 0.21933938562870026, -0.6279654502868652, -0.06576847285032272, 0.8391033411026001, 1.316625714302063], [-0.9258018136024475, 0.7453300952911377, -0.7479433417320251, -0.1511736661195755, -0.8888847231864929, 0.0835358053445816, 0.04968646541237831, 0.35568711161613464, -0.9645388126373291, -0.9416454434394836, 0.1705065369606018, -0.6453060507774353, 0.9772001504898071, 0.15997518599033356, 0.1255301833152771, -1.8589000701904297, -0.10001667588949203, 0.5168284773826599, -2.325120449066162, 0.08715158700942993, 0.2888321280479431, -0.23709018528461456, -1.464246392250061, -0.7452138662338257, 0.06964156776666641, 0.8299271464347839, -0.37824156880378723, 1.2474136352539062, -2.3878324031829834, -1.7314212322235107, -0.605253279209137, -1.21511709690094, -0.9127391576766968, -2.2843737602233887, -0.9781044125556946, 1.1502870321273804, 1.5422897338867188, -1.94627046585083, -0.27365127205848694, -0.7834264636039734, -1.2886993885040283, 3.652784824371338, 1.8363219499588013, 0.36924171447753906, 0.5972329378128052, 1.5861916542053223, -0.13794444501399994, 0.3046022653579712, -0.48233726620674133, -1.3042018413543701, 1.0623674392700195, 0.6589610576629639, 0.9601899981498718, -1.6640146970748901, -0.20913828909397125, 0.21348024904727936, -1.2470779418945312, -0.40533772110939026, -0.1722433865070343, -0.8712415099143982, -1.2622829675674438, 0.9589522480964661, -1.4615652561187744, 0.06829369813203812, -1.7633346319198608, -0.3546220362186432, 0.06314671784639359, -0.010435514152050018, 1.44383704662323, 0.30809485912323, -0.878699541091919, -2.4532217979431152, -1.15933096408844, -0.226925328373909, 0.8298043608665466, 0.12691478431224823, -0.22029195725917816, 0.5013371706008911, 0.6902851462364197, -1.2222583293914795, 0.49464914202690125, 0.1891889125108719, 3.214515209197998, -0.8369815945625305, 0.1361602395772934, 0.4843536615371704, 0.005578381009399891, -1.1416282653808594, -1.8459529876708984, -2.2103805541992188, 1.5320971012115479, -0.2651122212409973, 1.3941760063171387, -0.7014652490615845, 1.277225136756897, -1.672487735748291, 0.08014381676912308, -0.5036567449569702, -0.8954418897628784, -0.4108841121196747, -1.3719093799591064, -0.2951678931713104, 0.20530813932418823, 0.23351995646953583, -0.37169384956359863, 0.6958168148994446, -1.0963447093963623, -0.3521628677845001, 0.0571015402674675, 0.9459565877914429, -0.667320191860199, -0.7150914669036865, -0.3362846374511719, -1.7354401350021362, 0.7449601292610168, 0.8260166645050049, 0.8950182795524597, 0.12075911462306976, 0.6555850505828857, 2.3638460636138916, 1.2178767919540405, 0.006517596542835236, 0.11981307715177536, 0.9935739040374756, 0.5003350377082825, 1.2661099433898926, -1.2940841913223267, 0.7891700267791748], [-1.083466649055481, 0.41245076060295105, -0.30474862456321716, 0.08486954867839813, -0.6401517987251282, 0.6023409962654114, 1.7551424503326416, 0.44725897908210754, -0.7762457728385925, -1.023154854774475, 0.37569963932037354, 1.3335857391357422, 0.18237555027008057, 0.8630683422088623, -1.098405122756958, 0.8463390469551086, -0.16172859072685242, -0.5229173302650452, -0.9694331288337708, -1.0614374876022339, 0.6931996941566467, 0.1370846927165985, -0.27625763416290283, -0.27680066227912903, -0.46820858120918274, 0.6938672661781311, 0.47901690006256104, 0.4609850347042084, 1.543866515159607, -1.2829238176345825, 1.172114372253418, 0.005610391031950712, 0.1397198736667633, -1.9022107124328613, 1.2661638259887695, -1.143760085105896, -0.7966916561126709, -0.946739137172699, 0.06136200577020645, -0.7728052735328674, 0.6556852459907532, -0.6032726764678955, 0.8003679513931274, -0.29514241218566895, 0.48021793365478516, 0.38096484541893005, 0.33178406953811646, 0.37361669540405273, -0.2795178294181824, -0.36682751774787903, -0.48571011424064636, 1.494938850402832, -1.5778923034667969, 0.24528682231903076, -0.20849065482616425, -1.507204532623291, -0.7838833332061768, -1.476050615310669, 0.4491197466850281, 0.5142097473144531, -1.3036860227584839, -1.5491766929626465, -0.5521824359893799, -0.38058745861053467, -0.5702511668205261, -0.7317973971366882, -1.2059048414230347, 0.4024665951728821, 0.37127768993377686, 0.67927485704422, -0.7502949237823486, 1.1201497316360474, 0.09088165313005447, -1.5044832229614258, 2.2387259006500244, 0.046747881919145584, 0.31423380970954895, 1.136379599571228, -1.0610589981079102, 0.1805715262889862, -0.26328298449516296, -0.7965911626815796, 1.5842137336730957, 1.9697755575180054, 0.7795166373252869, -0.6216588616371155, -0.7145920991897583, -0.9586138725280762, -1.0141122341156006, 1.6089808940887451, 2.0019280910491943, -0.39478233456611633, 2.06284236907959, -2.0082757472991943, -1.1909111738204956, 0.3614574372768402, -0.9987539052963257, 0.9056900143623352, 0.5387725234031677, 0.026587212458252907, -0.035902973264455795, -1.6085233688354492, 1.604175329208374, -0.7770441770553589, 1.657763123512268, -2.0814247131347656, 0.45214930176734924, -0.36292359232902527, -0.7451586723327637, -0.4245636463165283, -0.5613599419593811, -2.1125266551971436, -0.45445460081100464, 0.9569663405418396, 0.13739833235740662, 0.2667091488838196, -0.32099586725234985, -0.7644869685173035, -0.5221232771873474, -1.194640874862671, 0.14676791429519653, 0.1845475435256958, 1.0643980503082275, -0.19979362189769745, 0.26775792241096497, -1.8104541301727295, -0.6897791028022766, 0.1790628433227539], [0.902062177658081, 1.1180663108825684, -0.5635412931442261, 0.09204874187707901, 0.028015341609716415, 0.21295997500419617, -0.39112669229507446, 0.941992998123169, -0.5093554258346558, -0.40142980217933655, -0.2166198492050171, 0.2841435670852661, 0.6043798923492432, 1.4133124351501465, 0.2973203659057617, 1.6884709596633911, -1.013494610786438, -0.253693550825119, -1.3128962516784668, -1.6159261465072632, -0.6496108770370483, -0.6973762512207031, -0.20087575912475586, -0.14024749398231506, -0.424021452665329, -1.9596240520477295, -0.16316856443881989, 0.5600936412811279, 0.19636943936347961, -0.46986323595046997, -0.6233101487159729, -0.7654959559440613, -0.5636140704154968, -0.21877603232860565, -1.2810221910476685, 0.33476385474205017, 0.9689939618110657, -0.40640565752983093, 0.9247424602508545, -2.384192943572998, -0.9818882942199707, 0.8178919553756714, -0.3951044976711273, 0.1393093764781952, -0.24950413405895233, 0.2880459427833557, -1.0263320207595825, 0.6533238291740417, -1.3759479522705078, 0.797831118106842, 0.7408795356750488, 0.39077794551849365, -0.31545382738113403, -0.6051350831985474, -0.05480601266026497, 0.5422443151473999, 1.6613761186599731, -0.6383312940597534, 0.4519702196121216, -0.05824485048651695, -0.4333164691925049, -0.48517856001853943, 0.4341345429420471, -1.1917524337768555, -0.7748887538909912, -0.6490197777748108, 1.0936756134033203, -1.2035678625106812, 1.518420934677124, 0.9012187123298645, -0.3745622932910919, 0.5961492657661438, -0.21467822790145874, 1.0793352127075195, 1.1577234268188477, -1.1488454341888428, 1.5175596475601196, -0.4964780807495117, 0.3117784857749939, -1.5624933242797852, -0.75553297996521, 0.35165664553642273, -0.957210898399353, -0.5085762143135071, -0.5149138569831848, 1.517225742340088, 2.7367255687713623, -0.9329893589019775, 0.2648203372955322, -1.0009351968765259, -2.186051845550537, -0.48564648628234863, 0.02720114402472973, -0.37736645340919495, -0.37229737639427185, 0.8066195249557495, -1.1785924434661865, -0.6137113571166992, -0.9677262902259827, -0.9082260727882385, 0.9475095272064209, -1.9403233528137207, 0.10285356640815735, 0.03520740196108818, 0.45327824354171753, -2.3853023052215576, -1.630932092666626, 1.12734854221344, -2.3434102535247803, -0.1375761479139328, 0.464125394821167, -2.5417115688323975, 0.4301947057247162, 1.0248513221740723, -2.122265100479126, -1.4702143669128418, 1.194297432899475, -0.9389809966087341, 0.8018950819969177, -0.3240561783313751, -2.0579421520233154, -0.9603367447853088, 1.1335171461105347, 1.5070273876190186, -1.0619118213653564, 1.0732442140579224, 0.5316171050071716, -0.18741166591644287], [0.9404513835906982, -0.6696039438247681, 0.9144638180732727, 0.18118242919445038, 1.0554661750793457, -1.6418805122375488, -0.002680805278941989, -0.12439475208520889, 1.080740213394165, 2.5475971698760986, 0.7883082628250122, 1.1349847316741943, 0.2806272506713867, -0.10223424434661865, 0.7364910840988159, 1.1261827945709229, 1.803293228149414, -2.0283334255218506, -0.7161402106285095, -0.0035189089830964804, 1.2448093891143799, -0.21615706384181976, -1.6234982013702393, -1.5583558082580566, -1.933296799659729, -1.9575581550598145, 0.6826256513595581, 1.122220754623413, -0.5354470610618591, -0.529900848865509, -1.4400957822799683, -0.8890421986579895, -0.14221671223640442, 0.14864511787891388, 0.5840739011764526, 1.8109923601150513, 1.0220539569854736, -2.1043848991394043, -0.7077109217643738, -0.5559590458869934, -0.5291451811790466, 0.5455774068832397, 1.358083724975586, -0.1848398894071579, 1.014345407485962, -0.4092385172843933, -0.6812956929206848, -0.7189990878105164, 1.2259327173233032, 0.24441614747047424, -1.117154598236084, 0.9449842572212219, -0.47300201654434204, -0.011032244190573692, 0.07913072407245636, 0.1677297055721283, 0.4023098349571228, -1.1581276655197144, -0.180304616689682, 0.8758583664894104, 1.879490613937378, -0.8536171317100525, -0.02121628075838089, -1.7288950681686401, -0.45525243878364563, 1.4557515382766724, -0.4902852475643158, 0.18176141381263733, -0.0593746155500412, 0.9283919334411621, 0.009636392816901207, 0.7406389117240906, -0.17619523406028748, 0.2014879286289215, -0.04572094604372978, -0.9592661261558533, 0.6225705146789551, -0.9916889667510986, -0.9920145869255066, -0.896285891532898, -0.11859698593616486, -0.6228218674659729, 0.200749471783638, 0.5398014783859253, 0.28174644708633423, 0.8388328552246094, -0.06596728414297104, 0.8047500848770142, -0.9371095299720764, -0.42496636509895325, -0.424410343170166, -0.17260468006134033, 0.3747439384460449, 1.043106198310852, 0.7694196105003357, 2.2297441959381104, 0.4686475396156311, -0.1558614820241928, 0.3701018989086151, 0.850459635257721, 0.4533098042011261, -0.4773571789264679, 0.8376898169517517, 0.47241368889808655, -0.8189998865127563, 1.1459505558013916, -0.9794555902481079, -0.4965537190437317, -1.4931159019470215, 1.0644865036010742, 0.04253407567739487, -2.186537742614746, 0.8028876185417175, 0.6516212821006775, 0.550905704498291, 0.8700743317604065, 1.7488455772399902, -0.9258606433868408, -1.0030068159103394, 0.25549620389938354, -0.8907393217086792, 2.5177812576293945, -0.857945442199707, 1.0380803346633911, -0.23796327412128448, -1.422847032546997, -0.6114303469657898, 0.246027410030365], [0.41961127519607544, -0.020550601184368134, -1.1832693815231323, 0.9178754687309265, -0.04627080261707306, 0.26679331064224243, -2.014244318008423, 0.9956936240196228, 0.9372719526290894, -1.137536883354187, 0.077670618891716, 1.1085319519042969, -1.1087632179260254, -0.75975501537323, -1.946031928062439, 1.9111474752426147, 0.6634350419044495, 0.3897455632686615, 0.42097947001457214, 0.5318268537521362, 0.5048468112945557, -0.6672182679176331, 0.9850485324859619, -1.5839916467666626, 0.2709123492240906, 1.1045054197311401, 1.8704650402069092, -0.445751816034317, -1.3003273010253906, -0.10218959301710129, 0.37918123602867126, 0.9838548898696899, -0.7466245889663696, 0.8015487194061279, 0.7279331684112549, 1.0779954195022583, 1.178902506828308, -0.6144586205482483, -0.02154441736638546, -1.2621536254882812, 1.4384735822677612, 2.1037089824676514, 1.3634099960327148, -0.7375184893608093, -1.5353171825408936, 0.21421581506729126, 0.5487058758735657, 0.5352146029472351, 1.6426821947097778, 1.253835916519165, -1.6104363203048706, -0.8868870735168457, -1.0986546277999878, 0.1650010049343109, 0.40821573138237, -0.9821721911430359, -1.0796884298324585, 0.837393045425415, 1.2912914752960205, -1.000857949256897, 1.1725069284439087, 0.0727815255522728, -0.8557618856430054, 0.10401702672243118, -1.6422313451766968, -0.009844549000263214, -0.18095943331718445, 0.7984799146652222, -0.17505241930484772, -1.0124412775039673, 0.6891806721687317, 0.3770461082458496, -0.9825286269187927, 0.7246379256248474, 2.890636444091797, 0.1886642873287201, -1.2245742082595825, -1.1834033727645874, 0.8208039402961731, 1.3386532068252563, -0.5304070115089417, -0.20642532408237457, 0.6373718976974487, -0.013273252174258232, -0.2811596095561981, 0.4769556522369385, 0.20329153537750244, -0.5553554892539978, -1.4470661878585815, -0.11341017484664917, 2.203173875808716, -0.3048880100250244, -1.544276237487793, 1.2724511623382568, 1.5844212770462036, 0.623268723487854, 0.16467706859111786, -1.0350531339645386, -2.3547823429107666, -0.11177239567041397, -0.1297294795513153, 0.13905566930770874, 0.4846690893173218, -0.6218752264976501, 1.1654733419418335, 0.28632691502571106, -1.447062373161316, 0.49752089381217957, 1.2994881868362427, 0.004940879065543413, -1.604668378829956, 0.3161548674106598, -1.0069079399108887, -0.5197027921676636, -0.31187382340431213, -1.1515257358551025, 0.2067655771970749, -1.6027584075927734, 1.0394022464752197, 0.7995322942733765, -0.3863550126552582, -0.9173177480697632, -0.28930315375328064, 0.7349114418029785, 0.3367517590522766, -0.926121175289154, 0.3610716760158539, 0.8632071614265442], [-1.4006812572479248, -2.5285539627075195, -0.7031235694885254, 1.5577974319458008, -0.6477392911911011, 0.12211231887340546, -0.7347192764282227, 1.2906763553619385, 0.00013320775178726763, 0.005550843197852373, -0.07960241287946701, 0.9555078744888306, -0.22995638847351074, 0.6546006202697754, 0.7089540362358093, -0.28825509548187256, 0.01551004871726036, -0.012479974888265133, -0.10112307965755463, -1.357733130455017, 0.037508632987737656, -0.6723328232765198, 0.22181247174739838, -0.5945612192153931, 0.3931737244129181, -0.1102631688117981, 1.309531569480896, -1.3831766843795776, -0.31292739510536194, 0.5218496322631836, -0.47742781043052673, -3.653031587600708, 0.1274293065071106, -1.698256015777588, 1.4958454370498657, 0.04474148526787758, -0.8375721573829651, 0.9315623641014099, 2.1161017417907715, 0.8288314342498779, -1.0368082523345947, 1.4775135517120361, -0.16792236268520355, -1.5362752676010132, 0.7989965081214905, 0.6041649580001831, -0.9614294767379761, -0.7553507089614868, 0.3291189968585968, -1.5658220052719116, -0.12738355994224548, 3.866520643234253, 0.3258928656578064, 1.069357991218567, 0.8630348443984985, -0.5257994532585144, -0.550861656665802, -0.3240438401699066, -0.35236889123916626, -0.3056611120700836, -0.5459573864936829, 1.319994330406189, 0.7260675430297852, 0.8871457576751709, -0.05375175550580025, -0.6200947165489197, 0.5639743804931641, -1.098748803138733, 0.9620282053947449, 0.3087821304798126, 0.9724360108375549, -0.2804926633834839, 0.022380178794264793, 0.23186303675174713, -0.33081814646720886, 0.40929779410362244, 0.04763432964682579, 0.16560973227024078, 0.717654287815094, -0.6242541670799255, 1.8928093910217285, -1.246741771697998, 0.6297621726989746, -0.5517261624336243, -0.4531334340572357, 1.443579912185669, 0.1755598783493042, -0.8298792839050293, 0.2822388708591461, 0.4075991213321686, 0.4907941520214081, 1.5307395458221436, -0.07653732597827911, 1.1098381280899048, -0.07006163150072098, -1.1502337455749512, 2.3690271377563477, 2.041062593460083, 1.2236289978027344, -0.7401612401008606, 1.0700876712799072, -0.4555218815803528, -0.0015093308174982667, -0.9068090319633484, 0.8516572713851929, 0.9075549840927124, -0.6121871471405029, -0.24279659986495972, -0.04771443456411362, 0.7796704769134521, -1.281032681465149, -1.5765743255615234, -0.8384057283401489, 1.1261078119277954, -1.3544927835464478, 0.05403420701622963, 0.5326566696166992, 0.6253364682197571, -0.5079839825630188, 0.24487993121147156, -2.408005714416504, -0.10110951960086823, 1.55888831615448, -0.6421406865119934, 0.6401845812797546, -0.28210607171058655, -0.15161938965320587, 0.22821259498596191], [0.7518799304962158, -0.4689764678478241, 0.4093177616596222, -1.031588077545166, 1.5460715293884277, -0.4351431131362915, 0.23635220527648926, -0.8259977698326111, -0.7191938757896423, -2.0812668800354004, -1.1213651895523071, 0.02663501910865307, -0.4543096125125885, 1.6426688432693481, 0.5268880724906921, -0.8206309080123901, 0.32051584124565125, 1.3894643783569336, -0.014012088999152184, -0.06596928089857101, 0.7435781955718994, -0.2643091082572937, 0.7042590379714966, -1.1486098766326904, 0.9057807326316833, -1.4956331253051758, 0.12804050743579865, 1.5431551933288574, 0.07091818004846573, 0.22353874146938324, -0.8291012048721313, -0.3801538646221161, 0.5643236637115479, 1.6446363925933838, -0.026191383600234985, 1.6114614009857178, -1.281548023223877, -2.184676170349121, -1.4710817337036133, 0.35415831208229065, -0.984006941318512, -1.7600351572036743, 0.5984064340591431, -0.29542115330696106, -0.8846665620803833, -0.4108662009239197, -0.3823724389076233, 1.0310719013214111, -1.2993534803390503, 1.0792042016983032, 0.027167120948433876, 1.0299910306930542, 0.9028269052505493, -1.2782021760940552, 0.15147371590137482, -1.016635775566101, -1.2280491590499878, 0.39730021357536316, -0.3163796365261078, -0.2536153495311737, 0.8679581880569458, 2.492983102798462, -1.693945050239563, -0.41154465079307556, -0.3422507345676422, -0.05060511454939842, 0.6577377915382385, -0.0730310007929802, -1.915209174156189, 0.2475898414850235, 0.4487723708152771, 0.30548882484436035, -0.9701896905899048, -1.1060810089111328, -0.3079942464828491, -0.4801176190376282, -0.7910590767860413, -0.48510006070137024, -1.1105844974517822, 0.606666088104248, 1.5432909727096558, 0.28363868594169617, 0.2497396320104599, -1.0006929636001587, 1.4317954778671265, -1.6787282228469849, 0.2937394976615906, 0.5574087500572205, -1.3131210803985596, -0.23411637544631958, -0.4597451090812683, 0.4041261076927185, 0.8469780683517456, -0.19079576432704926, -0.2871488034725189, 1.9245452880859375, -0.7972925901412964, 1.365817666053772, -0.3414109945297241, 2.4433956146240234, -0.6003059148788452, -0.4148734509944916, 1.5986961126327515, -0.3071220815181732, 0.5083984732627869, 0.7437946200370789, 0.19036084413528442, 1.2450522184371948, 0.21456462144851685, 1.2197468280792236, -2.0098867416381836, -0.15034453570842743, 1.512665033340454, -1.8765017986297607, 1.5074572563171387, 0.5558115839958191, 0.7296565175056458, -1.4640085697174072, -0.23871593177318573, 0.29175546765327454, -1.3529345989227295, 0.26051732897758484, 0.4435264766216278, 0.34964561462402344, -0.9725136756896973, -0.6899081468582153, 1.576642632484436, 0.7510724663734436], [0.018370093777775764, 0.981347918510437, 1.1918978691101074, 0.3411658704280853, 0.5468876957893372, 1.212722659111023, -0.576922595500946, 0.09786520898342133, -0.04503894969820976, 0.5302399396896362, -2.248276948928833, -0.7314016819000244, 1.2652403116226196, -0.6185941696166992, -0.6392077803611755, -0.23545047640800476, -0.3352416157722473, -0.7570230960845947, 1.7761179208755493, -0.16121654212474823, -2.9846560955047607, 0.6908182501792908, -1.1893556118011475, -0.9710906744003296, 0.6905468106269836, -0.2741866111755371, -0.30826088786125183, -0.7773504257202148, 1.4349799156188965, 0.7104278802871704, -0.5734219551086426, -0.24672174453735352, 0.5830478072166443, 0.08383975923061371, 1.0099502801895142, 1.5961681604385376, 0.07977856695652008, 1.016357421875, -2.4258549213409424, -1.7113701105117798, 1.2826257944107056, -0.6303467750549316, 0.7002348303794861, 2.254878044128418, 1.7302501201629639, 0.8602017164230347, 0.9108095765113831, -0.6747487783432007, 0.3979474902153015, 0.11001951992511749, 0.7481228709220886, 1.1539506912231445, 1.1427665948867798, -0.18005456030368805, 0.2927878499031067, 0.8485921025276184, -0.14189459383487701, 0.22369027137756348, -0.5539240837097168, 0.4912218153476715, 0.3618318438529968, 0.5334158539772034, -0.34987136721611023, -1.0077588558197021, 0.40139755606651306, 0.257750004529953, 0.74993497133255, 0.08153661340475082, 1.5755455493927002, 0.7393434643745422, -1.392289638519287, -0.13034558296203613, -0.28909438848495483, 0.9178543090820312, -0.011135688982903957, -1.3253324031829834, 1.2798941135406494, 0.5250806212425232, -1.8129717111587524, -0.5524746775627136, -0.9178186655044556, 0.7951949834823608, -0.5689409971237183, 0.6525307893753052, 0.2964743673801422, -2.101473808288574, -2.2007431983947754, 0.4372819662094116, 0.2687051594257355, 0.9973447322845459, -1.1382900476455688, 1.801936388015747, -1.0202805995941162, 0.4254268705844879, -0.5121206045150757, -0.1531641185283661, -0.2163529098033905, -0.7072530388832092, 0.7359932065010071, 0.7816153168678284, 1.2937812805175781, 0.4683188199996948, 0.6333878040313721, -0.5231876969337463, 0.6209057569503784, 0.9947025775909424, -0.32257434725761414, 1.2756901979446411, 0.2903549075126648, -0.7024857401847839, -0.0945628434419632, -1.2683552503585815, 0.7531015872955322, 0.08904925733804703, -0.014360075816512108, 0.2231101244688034, -0.8284494280815125, 1.0869866609573364, -0.747132420539856, 0.40858015418052673, 0.3219620883464813, -0.1276070773601532, 0.14025025069713593, -0.24261920154094696, -0.6395967602729797, 0.22784318029880524, 1.4625555276870728, -1.032355546951294], [0.6558860540390015, 1.174971342086792, 0.46655336022377014, -0.8438834547996521, -0.07037888467311859, -2.319164514541626, 1.0671558380126953, -0.37348097562789917, -0.5687845349311829, 0.8930564522743225, 1.32789945602417, -1.020944356918335, 0.5887264013290405, -0.5035668611526489, 0.5242671966552734, -0.6370870471000671, -0.6230701208114624, 0.8876034021377563, -0.32451990246772766, -1.1861965656280518, -0.9615036845207214, -0.6909343600273132, -0.7132254838943481, -0.6372560262680054, -0.5158503651618958, 0.3752545416355133, -1.9275224208831787, -0.14140449464321136, -1.027404546737671, 0.4161900281906128, 0.1548735648393631, 0.4845130741596222, 0.1886209100484848, -0.6543164253234863, 0.5495124459266663, -0.1428484469652176, -0.7991943955421448, 0.918725848197937, -1.091775894165039, 0.9472223520278931, -1.6607249975204468, -0.075648732483387, -0.60762619972229, -1.5554816722869873, -0.45823681354522705, 0.5162692070007324, -0.396028995513916, 1.291154384613037, 1.3487190008163452, 0.8649289608001709, 1.4278569221496582, -1.6161727905273438, -1.2092491388320923, -0.2187834084033966, 0.9805787205696106, -0.36321884393692017, 0.4069141447544098, 1.7117488384246826, -1.0540471076965332, 2.7972469329833984, -0.270043283700943, -1.0085924863815308, 0.3651098906993866, -0.9970203042030334, -2.5185775756835938, -2.6271495819091797, -0.2819182872772217, 0.19743677973747253, -2.8844361305236816, 0.6930203437805176, -0.7969970703125, -0.6910919547080994, 0.5311597585678101, -0.5830419063568115, -1.1843514442443848, -1.474183440208435, -0.16576805710792542, 0.781015932559967, 1.2181447744369507, -0.48340150713920593, 1.8734387159347534, 2.3151683807373047, -2.080519914627075, -0.5146113038063049, -0.43068450689315796, 0.2974366247653961, 0.0748402327299118, 0.03568238392472267, -0.0494028739631176, 0.047550003975629807, -0.6009352207183838, 0.14154835045337677, -0.1255568563938141, 0.5473344922065735, -1.5365928411483765, 0.44883543252944946, -0.7561939358711243, 0.882566511631012, -0.5943854451179504, 1.3333241939544678, 0.21203091740608215, -1.4096381664276123, -0.23601599037647247, -1.8378576040267944, 2.312307357788086, -0.4760018289089203, -1.7932970523834229, -0.6480270028114319, -0.6333038806915283, 0.5187981724739075, 0.8419368863105774, 0.6913281679153442, -0.7071930170059204, -0.6396369338035583, -0.9913558959960938, -1.0342086553573608, -0.40212002396583557, 0.21092352271080017, -0.3709118664264679, -0.05256282165646553, -1.0147249698638916, 0.6558261513710022, -0.5906877517700195, -0.6593948006629944, -0.2809579074382782, 0.0499100387096405, -0.2345544397830963, 1.0292954444885254], [1.2146618366241455, 0.1406034380197525, -0.08255708962678909, -0.36404672265052795, 0.39278310537338257, -0.5572306513786316, -0.12074550241231918, -1.3128048181533813, -0.09207146614789963, -1.2143609523773193, -0.5817254185676575, 0.4701248109340668, 1.4408352375030518, -0.48406657576560974, 0.3969576060771942, 1.1164830923080444, 1.4964038133621216, -0.8993239402770996, -0.2088085263967514, -2.131053924560547, 2.2138893604278564, -2.2766339778900146, -0.14710400998592377, -0.4486640393733978, -0.35366132855415344, 0.943432629108429, 1.4908794164657593, 1.2519420385360718, 0.8250814080238342, -0.8093432784080505, 2.3943560123443604, 0.6445341110229492, -0.6984551548957825, -1.270682692527771, -0.10094040632247925, 3.120651960372925, 0.25573110580444336, -0.2513105869293213, -0.25168561935424805, -2.1436355113983154, 0.010299254208803177, -0.5980501174926758, 0.1318025141954422, -2.1511940956115723, 0.7497549057006836, -0.13191765546798706, -0.3669810891151428, 0.3922550678253174, -0.6090688109397888, -0.42193567752838135, 1.9141069650650024, -0.05144686624407768, 0.8293282389640808, -0.5826703310012817, -0.6959807872772217, -0.05405318737030029, 0.7916380763053894, -0.3738202750682831, -1.301410436630249, 1.4807510375976562, 1.9902530908584595, 0.12747429311275482, 0.023356765508651733, -0.50437331199646, -0.6682530045509338, -1.3872393369674683, 0.6474326252937317, 0.7047035098075867, -0.7344654202461243, -0.4441177248954773, -0.5358996391296387, -0.8387807607650757, -1.3963985443115234, -0.064698226749897, -0.4433014988899231, 0.04103961959481239, 0.8533671498298645, 0.7398542165756226, 0.015704309567809105, 0.1709500551223755, -1.2390280961990356, -0.7152327299118042, 0.5581124424934387, -0.5726686120033264, -0.20179316401481628, -0.24390986561775208, 0.29739752411842346, -0.06739561259746552, -2.144437789916992, 0.9756144881248474, 0.5682201385498047, 0.08684837073087692, 0.47430065274238586, -0.2726074159145355, -0.33938074111938477, 1.3194152116775513, 0.5351572036743164, 1.9353270530700684, 0.6633940935134888, 0.14320065081119537, -0.3333892822265625, 0.04949120804667473, -0.4509100019931793, 0.2425585836172104, 0.5975503921508789, -0.21034149825572968, 0.9352289438247681, 0.7358525395393372, 1.017276644706726, -0.7930224537849426, 1.3497989177703857, 2.6848413944244385, -1.1194138526916504, 0.660205066204071, 0.9577228426933289, -0.3735068440437317, 1.3959965705871582, -0.07680169492959976, -0.5733289122581482, 1.1564571857452393, -1.746058702468872, -0.0389213040471077, -0.3055468499660492, -0.4499782919883728, 0.41316965222358704, 0.5912085771560669, 1.3767086267471313, -0.12405235320329666], [0.49907708168029785, -0.13148289918899536, -0.5222135782241821, -0.9755010008811951, 0.9909220337867737, 1.189552903175354, 0.33314165472984314, -0.40315040946006775, 0.029547257348895073, -0.41916513442993164, 1.1777862310409546, -1.6553246974945068, 1.4615031480789185, 0.1424494832754135, 1.0033711194992065, -0.14559635519981384, -1.8320820331573486, 2.02626895904541, -0.7282776832580566, -0.5411430597305298, -0.5113601684570312, 0.7299449443817139, 0.712398886680603, -0.8783344626426697, -1.2212989330291748, -0.41575390100479126, -0.014614484272897243, -0.2465437650680542, 0.33389610052108765, -0.3185862898826599, -1.0035027265548706, 1.274870753288269, 1.1215510368347168, 0.5077656507492065, -1.941635012626648, 1.6161599159240723, -1.1685640811920166, 0.6643678545951843, 2.039663553237915, 0.16571833193302155, 0.04430393502116203, 1.6562381982803345, -0.7573091983795166, 0.3639833331108093, 0.045193858444690704, -1.09992516040802, 1.9198508262634277, 0.3607173562049866, 0.27865850925445557, -0.9733843803405762, -0.8246891498565674, -0.4321768283843994, -0.493726521730423, -0.18683640658855438, 2.031385898590088, 1.363128423690796, -0.117239810526371, -0.24799545109272003, 1.8052362203598022, 1.6665689945220947, 0.05467066913843155, 0.4184550344944, 0.29663407802581787, -0.7024971842765808, 0.8050379753112793, 0.50279700756073, -0.12965907156467438, 0.32097846269607544, 1.0735286474227905, -0.3674105703830719, 2.024596929550171, -2.0248963832855225, 0.6443231701850891, 1.3033361434936523, -2.179699420928955, -0.04624967277050018, -2.1183154582977295, -1.7398375272750854, 0.6223823428153992, -0.47428855299949646, 1.6452655792236328, -1.0737468004226685, 0.3406449854373932, 0.44080281257629395, -0.7889440655708313, -1.5988664627075195, -2.034615993499756, 0.8608916401863098, 1.9932290315628052, -0.222802996635437, -0.7890612483024597, -0.4495091438293457, 0.8937336802482605, -0.3763517141342163, 0.11786264181137085, -1.3614351749420166, 0.27775251865386963, -0.6394072771072388, -1.3034229278564453, 0.2966209948062897, 1.3407713174819946, -0.0763750672340393, 0.011056870222091675, -0.29267996549606323, -0.854263961315155, 0.22017760574817657, -0.22257712483406067, 1.493971824645996, -0.7919903993606567, -1.8463290929794312, 1.7284824848175049, 0.670827329158783, 0.38957586884498596, -1.2681264877319336, 0.5727189183235168, -1.194061279296875, -1.6531611680984497, 0.4757082462310791, -0.9410219788551331, 1.3026585578918457, -0.14118725061416626, 0.17769168317317963, 0.8913666009902954, -1.6482758522033691, 2.054534435272217, -1.7955824136734009, 0.7562653422355652, 0.13924169540405273], [0.16247646510601044, -0.3185007870197296, 0.651008665561676, 0.3041408061981201, -0.35582250356674194, 0.31360140442848206, 0.5891240239143372, -1.8698716163635254, -0.42480504512786865, -1.388580560684204, -0.14442817866802216, -1.6174795627593994, 0.3797883093357086, -0.7884772419929504, 0.2269665002822876, -1.7781208753585815, 0.0129482327029109, 1.374318242073059, 0.33363625407218933, -1.5424790382385254, -0.36838844418525696, 0.9034259915351868, -1.5585548877716064, -0.47727787494659424, 1.211747407913208, -0.8271144032478333, -1.126164436340332, 0.182374969124794, 0.4834650456905365, 1.0707464218139648, 1.3572440147399902, 0.6155372262001038, -0.5916313529014587, -2.039015769958496, -0.7102518081665039, -0.08283548057079315, 0.740825891494751, -0.2564786374568939, 0.07438068836927414, 0.8018339276313782, -1.1663175821304321, -0.5339807271957397, -0.7314466238021851, -0.5832487940788269, 0.37660014629364014, 1.6420260667800903, 1.7919093370437622, 1.4345922470092773, 0.3322972059249878, -0.6732217669487, -0.6829022169113159, 0.9983729720115662, 0.5194718837738037, -0.5869448781013489, 0.6817993521690369, 0.287070631980896, 1.0860280990600586, -1.0571962594985962, 0.28045135736465454, 0.07366891205310822, 0.6518696546554565, -0.45777973532676697, 0.14507576823234558, 0.2521374523639679, 0.36660003662109375, 0.8390006422996521, -0.5816071033477783, 2.6316072940826416, -1.9106268882751465, 0.6368409991264343, 0.22329305112361908, -0.42605361342430115, -1.1461659669876099, 2.210855484008789, 1.2510855197906494, -0.7226350903511047, -0.38259226083755493, 0.17462629079818726, 0.3068437874317169, 0.2418387532234192, 0.874828577041626, 0.2700229287147522, 1.4660301208496094, -0.3609456419944763, 0.3037993013858795, -0.5839307904243469, -0.8597473502159119, -0.4409627914428711, 0.778715193271637, 0.4282072186470032, 1.5897350311279297, 0.4534217417240143, 1.0925791263580322, -1.1046029329299927, 0.048679906874895096, 2.8166565895080566, 1.3412141799926758, -0.06252215802669525, 0.9585594534873962, 0.4289894700050354, 1.97769296169281, -0.5407934784889221, 1.7449787855148315, 0.8815553784370422, -0.5839366316795349, 1.1513758897781372, 0.06409358233213425, 2.449246883392334, -0.9867829084396362, 0.4681706726551056, 0.02323685586452484, 1.9148766994476318, 0.06527242064476013, 0.3060680329799652, 0.35177356004714966, -0.2769688665866852, -0.7464302778244019, 0.04386477172374725, -0.4918768107891083, -0.3867512047290802, -0.27460166811943054, -0.6777183413505554, 0.00035954429768025875, -1.216381549835205, -1.15938138961792, -0.837783694267273, 0.03378111124038696, -0.0002770089195109904], [-1.7982381582260132, -0.07412106543779373, -0.0739273726940155, -0.4044070243835449, 0.9866806864738464, 0.7898138165473938, 0.9922500848770142, -0.11196759343147278, -0.7137539386749268, 1.0067192316055298, -0.6683311462402344, -1.3914768695831299, 1.1010159254074097, 0.4517515003681183, 0.07729707658290863, -0.37699270248413086, 0.25605538487434387, -1.3932749032974243, -0.04259004816412926, -0.22269611060619354, -1.623633861541748, -0.48245659470558167, -0.4505494236946106, -0.5912887454032898, -1.052377700805664, -1.115356206893921, 1.544451355934143, 0.4958398938179016, -1.1649377346038818, 0.5324690937995911, 0.0658905878663063, 1.4018162488937378, 0.9331876635551453, 1.87577486038208, 0.0757436603307724, -0.4480714201927185, -0.552569568157196, 0.8531579375267029, 0.9605311155319214, -0.6057400107383728, -1.4784609079360962, -0.3598374128341675, 0.06409069895744324, -0.06030138581991196, 0.18741583824157715, 1.2794846296310425, -0.38919419050216675, -0.014323040843009949, -0.22555434703826904, -0.9702747464179993, 0.43437108397483826, 0.08015379309654236, 0.12209053337574005, -0.18607649207115173, 0.07197783142328262, 0.607115626335144, 1.2551922798156738, -1.098522424697876, 0.9233614802360535, -0.013301244005560875, -0.20549942553043365, 0.6412755250930786, 0.3669413626194, 1.1888889074325562, -0.004269338678568602, 2.154836654663086, -1.1806883811950684, -1.078033447265625, -1.7000501155853271, -0.03514023870229721, -1.5877641439437866, 1.676276683807373, 0.649555504322052, -0.4508219063282013, 2.3754193782806396, -1.4756823778152466, -1.467233657836914, 0.07767633348703384, 0.4261743128299713, -0.6230730414390564, -0.0737794041633606, 0.4292415678501129, -0.16208723187446594, 0.6819891333580017, -1.434476613998413, -1.9124157428741455, 1.2222049236297607, -0.4182019531726837, -1.195033311843872, -0.39483368396759033, 1.3891587257385254, -0.8795579671859741, 0.12604781985282898, -0.11823461204767227, 0.13390497863292694, 1.5647813081741333, 1.7697848081588745, 1.1664339303970337, 1.2942512035369873, 0.09423190355300903, 0.596678614616394, -1.9692164659500122, 1.3337244987487793, -1.5212820768356323, -1.4520107507705688, -0.3837067186832428, -0.004597553983330727, 0.3011361360549927, 1.2709286212921143, 0.15225045382976532, -0.5241258144378662, -0.4711146950721741, 0.3955595791339874, -0.7618789076805115, 1.6031339168548584, 0.4460288882255554, 0.6723383665084839, -1.6217758655548096, -1.6356720924377441, 1.070359230041504, -0.44386231899261475, 0.6767390370368958, -0.1373448520898819, -0.11135950684547424, 2.1067922115325928, -1.5745781660079956, -0.7383987307548523, 2.095205307006836], [-1.7917295694351196, 0.06483094394207001, -2.4310672283172607, 0.08904213458299637, -0.37273040413856506, -0.5037652254104614, -0.0645654946565628, -0.4101739525794983, 0.38274523615837097, -0.006320103071630001, 0.46159452199935913, 0.8763110637664795, 0.7371581196784973, 0.5348299145698547, 0.9932968020439148, -1.066046118736267, -0.7355870008468628, -1.3393229246139526, 0.7069936394691467, -0.6795548796653748, 0.9408150911331177, 0.043866366147994995, -0.7507796883583069, -0.21586468815803528, 0.4320356547832489, -1.3833208084106445, -0.15915250778198242, 2.2989869117736816, -0.38758400082588196, -1.693739414215088, -1.7341854572296143, -1.217643141746521, -0.26485633850097656, 0.3709617257118225, -1.2395005226135254, 1.1148061752319336, 0.9303411245346069, 1.4536025524139404, -1.2740188837051392, -0.11610181629657745, -0.7803534865379333, -0.4033587872982025, 1.6964120864868164, 0.5003424286842346, -2.3869516849517822, 1.7352159023284912, -1.1688066720962524, -0.00013454009604174644, -0.2782997190952301, 0.2256341278553009, -0.899937629699707, -0.9424335956573486, -1.6172178983688354, -1.2605550289154053, -0.12361757457256317, -0.870994508266449, 1.644448161125183, -0.5509184002876282, -0.844875693321228, 1.7949858903884888, -0.6028797626495361, -0.8972577452659607, 0.5603628158569336, -1.3278712034225464, -0.13033345341682434, 0.7796822190284729, 0.7237809300422668, -0.026046769693493843, 1.362163782119751, 0.6209612488746643, 0.1610969752073288, -0.4039102792739868, -1.2854844331741333, -0.6698138117790222, 0.055918388068675995, 0.9891256093978882, 0.6955485939979553, 1.219967246055603, -1.588173508644104, -0.7714108824729919, -0.4086419343948364, -2.1944668292999268, -0.07320721447467804, 1.202987551689148, 1.5153394937515259, -0.14735578000545502, 0.09864180535078049, 0.6581971645355225, -0.10833572596311569, -0.46948227286338806, 1.6859363317489624, -0.6561222076416016, -0.5291541814804077, -1.0495879650115967, -0.46541351079940796, 0.7702771425247192, -0.12791386246681213, 0.8916618824005127, -0.1196703240275383, 0.2926722466945648, 0.3107839822769165, -0.5942073464393616, 0.7208015322685242, 0.6902475953102112, -0.815623939037323, 0.649850070476532, 0.5108404755592346, -0.6873063445091248, 1.3854455947875977, -0.11509236693382263, -0.7296644449234009, 0.09899970889091492, -0.6843710541725159, -0.13832339644432068, 0.6382832527160645, 0.4470524787902832, 0.26963791251182556, 0.6490927338600159, 1.7908964157104492, 0.9542490839958191, 1.5473016500473022, 0.8926109075546265, -0.49919524788856506, 0.2404126524925232, -0.3294731080532074, 0.5793271660804749, -2.1719870567321777, -1.2814213037490845], [0.5615072846412659, 1.0506742000579834, 0.616976261138916, 0.1995486617088318, 0.2874612510204315, -1.0578769445419312, 0.4787941873073578, 0.48708945512771606, 2.41495680809021, -0.15106990933418274, 0.20643320679664612, -1.2720704078674316, -0.41370394825935364, 1.723252296447754, 1.034469723701477, -0.7679234743118286, -1.4824614524841309, -0.5669673681259155, -1.3361129760742188, -0.0908636748790741, -0.771683931350708, -0.48010626435279846, 1.1295592784881592, 1.014573097229004, 0.4034918248653412, 0.012723609805107117, -0.28871530294418335, -1.5745793581008911, 0.771542489528656, -0.9551430344581604, 1.1573905944824219, -0.20590199530124664, -0.8619900941848755, -1.551652431488037, -0.08625559508800507, 0.18320263922214508, -0.8357205986976624, 0.32098889350891113, -1.1432071924209595, -0.23931919038295746, -1.038110613822937, -1.0761698484420776, -2.0281786918640137, -0.5812456011772156, -0.49089133739471436, -1.6559487581253052, -0.9409650564193726, -0.3548847436904907, -0.083468958735466, -0.41594111919403076, 0.3024732172489166, -0.40629255771636963, 0.3606663644313812, -0.5599465370178223, 0.8967660665512085, 0.04660810902714729, 0.593230664730072, -1.2872858047485352, 0.36387133598327637, 0.7544297575950623, -0.2802058756351471, -0.29058414697647095, 0.9549877047538757, -0.06009652465581894, -0.0913124829530716, -0.1830470710992813, -1.0039470195770264, 0.40551313757896423, -0.08211146295070648, -0.1140393540263176, 0.38634759187698364, 0.4484117925167084, 0.9149470329284668, -0.3391647934913635, -1.9764422178268433, -0.7307509183883667, -2.007197380065918, 0.5860897302627563, 0.5040369033813477, -0.3577090799808502, 0.04116043075919151, 1.8434652090072632, -0.7558480501174927, 1.845375895500183, 1.0093034505844116, 1.4997094869613647, -0.442462682723999, -0.26566046476364136, 0.32365426421165466, -0.7501962184906006, -0.32173997163772583, 0.09638041257858276, 1.1041882038116455, -0.5160609483718872, 1.1275441646575928, 0.999569833278656, 1.1928335428237915, 0.43789151310920715, -0.3485212028026581, -1.7851536273956299, 1.5655407905578613, 2.1658997535705566, -0.9908075332641602, 0.9290909171104431, -0.03126120567321777, -0.40476998686790466, -0.6166805624961853, -0.9380798935890198, -0.9542266130447388, -1.9340342283248901, -1.0280336141586304, 1.080325961112976, 0.029143622145056725, 0.5095189213752747, 0.5229045748710632, 0.38654306530952454, -0.2243587225675583, 0.7636846303939819, -1.8727750778198242, 1.513702154159546, -1.7484104633331299, 0.2376658320426941, -0.5930467844009399, 0.09369010478258133, 0.12602965533733368, 0.12208220362663269, 0.49931731820106506, 0.09826706349849701], [1.8674166202545166, -0.22533585131168365, -1.3261992931365967, -1.0870143175125122, 2.0019774436950684, 1.3651401996612549, 1.2893983125686646, 1.1983120441436768, 0.8457651138305664, -0.17556895315647125, -2.006157398223877, -0.2788422107696533, 0.13778872787952423, -1.210128664970398, -0.45136886835098267, 0.2686794400215149, 0.8950076699256897, 0.36237964034080505, -0.4167701005935669, -0.7194619178771973, -0.5004117488861084, 0.3598281741142273, -0.5133704543113708, 1.2133111953735352, -0.03808772191405296, -0.5262444019317627, 0.7518666982650757, 1.8231208324432373, 0.8156318664550781, 1.9297796487808228, -0.8524359464645386, 0.9837028980255127, -0.13536368310451508, 0.16202247142791748, 0.39769676327705383, 0.4543939232826233, 0.058097947388887405, 0.587812602519989, -1.1595523357391357, 1.3584808111190796, -0.5639973878860474, 1.874336838722229, 1.0730253458023071, 0.9151538014411926, -1.1424559354782104, 0.2652980387210846, -0.43956002593040466, -0.04837587848305702, 0.8148854374885559, -0.2797101140022278, -0.5091763138771057, 0.5058574080467224, -0.6845916509628296, 1.7884900569915771, -0.7948772311210632, -0.7124480605125427, 1.282702088356018, -1.315290093421936, 0.29505398869514465, 0.11587046831846237, -0.6176059246063232, -0.46887874603271484, 1.2370978593826294, 0.568032443523407, 0.451171338558197, 0.20944294333457947, 1.5968154668807983, -0.7306729555130005, 1.2216627597808838, -1.2162694931030273, -0.403584361076355, 1.0008262395858765, -0.4163056015968323, -1.7417691946029663, -1.320133924484253, 0.7012161016464233, 1.7886865139007568, -0.3572099208831787, 0.24369312822818756, -0.35990604758262634, -0.6031831502914429, -0.47618231177330017, -0.8885424137115479, 0.8206195831298828, 0.027197957038879395, 0.8795498609542847, 0.5388681292533875, 0.27854394912719727, -0.5852687954902649, 0.36293697357177734, -1.0754992961883545, 0.29467445611953735, 1.3490186929702759, 0.36832934617996216, 0.9257521629333496, 0.4600732922554016, -0.04106755182147026, -0.17316092550754547, -0.615425705909729, -1.4750162363052368, 1.5251964330673218, 1.8962230682373047, 0.2738991677761078, 1.764926791191101, 1.133993148803711, 2.0109753608703613, -0.5217019319534302, 0.2727113664150238, -1.3598158359527588, -0.440525621175766, 0.331449419260025, 1.7588895559310913, 1.284234642982483, -0.01067412830889225, -1.752625823020935, -0.21818557381629944, 0.43758368492126465, -0.03908411040902138, -0.41369348764419556, -0.6987738013267517, -0.5749267935752869, 0.8254644870758057, 0.6590964794158936, -0.6248636841773987, 0.18980278074741364, -1.8178014755249023, -1.4887983798980713, -0.994187593460083], [-0.27628999948501587, -0.7961910963058472, -0.12020129710435867, 0.19990979135036469, 0.3747826814651489, 0.6524173617362976, -0.6890771389007568, 1.3524951934814453, 1.5988367795944214, -0.2855072021484375, 1.1668018102645874, 2.7924575805664062, -0.6219186186790466, 1.3101955652236938, 0.6872367262840271, 0.09198088943958282, 0.5244566798210144, 1.6913915872573853, 1.3298290967941284, 1.2138442993164062, 0.2616356909275055, 0.02207142300903797, -0.3052167594432831, -1.7247320413589478, -0.3486361801624298, 2.2276413440704346, 0.17484666407108307, 0.15108969807624817, 1.2502468824386597, -0.40184691548347473, -0.6520903706550598, -0.5154424905776978, -1.002848505973816, 0.609235942363739, -0.21578925848007202, 1.58964204788208, -0.5285114645957947, -0.3394679129123688, 0.45611363649368286, 2.4246842861175537, 0.4846579432487488, 0.1480547934770584, -0.746086597442627, 0.5964305400848389, -0.5521214604377747, -1.8099726438522339, 0.7592115998268127, 0.619357705116272, 0.5813621878623962, -0.22353887557983398, -0.8453269600868225, 0.7752046585083008, -0.14295843243598938, 0.3408255875110626, -1.0857959985733032, -1.0407135486602783, 0.4848233163356781, 0.2379191517829895, 1.0825741291046143, 2.327531576156616, 1.0070104598999023, 0.5835101008415222, -0.24512438476085663, 0.8459981679916382, 0.7316925525665283, -1.3141604661941528, -2.351661443710327, 1.7185609340667725, 0.23811325430870056, -0.11175359785556793, -2.3483469486236572, -0.16462761163711548, -0.3771873712539673, -0.5233919024467468, 0.12093844264745712, 0.11815208941698074, -0.9851900935173035, -1.8628202676773071, 0.8326918482780457, -0.3818012773990631, -1.7791696786880493, 0.517676055431366, -0.2731570303440094, -0.9747467041015625, 0.03839637711644173, -0.3661532402038574, 0.150231271982193, -0.39342668652534485, -1.3193490505218506, 0.3467981815338135, -0.05525738373398781, 0.8859076499938965, -0.052485834807157516, 0.3765728771686554, -0.9110355377197266, 0.18174642324447632, 0.6744256615638733, 0.6891621947288513, 0.9181241989135742, 0.3522435128688812, 1.3237924575805664, 0.7265146970748901, 0.7409180402755737, 0.9746659994125366, 0.06029487028717995, -0.635033130645752, 1.104320764541626, 0.06937651336193085, 1.93808913230896, 0.1967795044183731, 0.12757596373558044, -0.09254901856184006, 0.5434157252311707, 1.266481876373291, -0.2664099335670471, -0.033606529235839844, 0.16834892332553864, 0.5836961269378662, -1.0084607601165771, 0.4577134847640991, 1.1101421117782593, -1.6365755796432495, -0.28717589378356934, -1.068824052810669, 0.38717207312583923, 0.41294044256210327, 0.9123286604881287, -0.20502343773841858], [-0.15722261369228363, -0.04678166285157204, -0.2798948287963867, -0.6323278546333313, 0.3437705636024475, -1.3836613893508911, -0.18300123512744904, 1.036324143409729, 0.5237505435943604, -1.0936681032180786, -1.2391886711120605, -0.9769372344017029, -0.3350508511066437, -0.9822789430618286, 0.9259611964225769, -0.3296446204185486, -1.6209757328033447, 0.05070750042796135, -1.1318016052246094, -0.09398680180311203, -0.03431856632232666, 0.7001729011535645, -0.20619598031044006, 0.1970236748456955, 0.17330989241600037, 1.537726640701294, -0.35007786750793457, -0.8581589460372925, 1.7419989109039307, -0.10246320068836212, -0.7706969976425171, 1.4485520124435425, 0.5140939950942993, -0.09182365983724594, 1.6478686332702637, 2.115093231201172, 1.8311376571655273, 1.4463140964508057, 2.754473924636841, 0.795714259147644, -0.932636022567749, -1.534851312637329, -0.3011370599269867, 0.24525360763072968, 0.9801250100135803, -1.2479422092437744, -1.0527724027633667, 1.959902286529541, -0.7818090915679932, 0.3282735049724579, 1.4945785999298096, 1.2865171432495117, -1.2737699747085571, 0.12874312698841095, 0.09164770692586899, 0.7816261053085327, -0.20127660036087036, 0.34547555446624756, 0.44788554310798645, 1.5201137065887451, -1.0721725225448608, 0.5463334918022156, 1.5692273378372192, -1.7281018495559692, 1.4454160928726196, -1.8904672861099243, -2.2027225494384766, -0.5790961980819702, 0.9263343811035156, 2.6567320823669434, -1.0594522953033447, -0.661349356174469, 0.09104105085134506, 0.5958113074302673, -1.5860121250152588, -0.7169954776763916, -0.13794848322868347, 0.2373836487531662, -1.049792766571045, -0.47799691557884216, -0.5354458689689636, 0.5003564953804016, -0.709123432636261, -0.1431157886981964, 0.1997690498828888, 1.194509506225586, -0.2739713788032532, 0.056617625057697296, -0.10799300670623779, -0.3286769986152649, 1.3911428451538086, 0.40134334564208984, 0.11178819835186005, 0.32494691014289856, -1.199664831161499, 0.3487158417701721, -1.0215892791748047, -2.004690408706665, -0.919614851474762, 0.43555504083633423, 0.5701364278793335, 1.0439521074295044, -0.6731203198432922, -2.08138108253479, 0.8772917985916138, -0.43574169278144836, -1.8790035247802734, 0.8363485336303711, 1.3154222965240479, -0.109518863260746, 0.029937580227851868, -0.3813423812389374, 1.0729517936706543, -0.13752123713493347, 0.517368733882904, 0.18206080794334412, -0.8391835689544678, 0.7187929749488831, 0.04928913339972496, 0.4686662554740906, -1.2045129537582397, -0.9943112730979919, -0.9960875511169434, 1.8899412155151367, 0.25669029355049133, -0.17811988294124603, 1.245408535003662, 1.0168166160583496], [1.1081753969192505, 0.6087381839752197, -1.0822752714157104, -1.5412359237670898, 0.2508206367492676, 0.27184978127479553, 0.20148149132728577, 0.44436517357826233, 0.7782109975814819, -0.07959269732236862, -0.4333135187625885, -1.4703166484832764, 0.11255152523517609, -1.2321674823760986, 1.081112027168274, 0.1996174305677414, 2.6697707176208496, 0.7014132142066956, 0.3216812014579773, -0.45333579182624817, 0.12472566962242126, -0.36998462677001953, 0.2567902207374573, -0.2508808970451355, 0.02057466097176075, -1.0614434480667114, 0.590806245803833, 0.889254629611969, -0.04233599826693535, 0.8969994783401489, -2.853726387023926, -0.4735185503959656, -0.7995765209197998, -1.3605108261108398, -0.7517557740211487, 0.3674510419368744, 0.3792896866798401, -2.756760835647583, -0.22530926764011383, 1.376479983329773, -0.8460785150527954, -0.05485072359442711, 0.16164293885231018, -1.944946527481079, -0.05429009348154068, -0.5753857493400574, -0.12444792687892914, -0.525462806224823, 0.5335217118263245, -1.0326628684997559, 0.857361376285553, 0.15452592074871063, -0.036416009068489075, 1.0776145458221436, -0.3412681221961975, 1.8177027702331543, 0.6710518002510071, 0.3164213299751282, 0.5725012421607971, -0.8406814336776733, 0.1821167916059494, -0.04041912034153938, -0.6646831035614014, 0.8813881874084473, 0.8332544565200806, 0.7048474550247192, -0.049139752984046936, 0.8125953674316406, 0.21025866270065308, 1.237254023551941, -0.20912709832191467, 0.6342015862464905, 0.11392059922218323, -1.0469865798950195, 0.5386143922805786, -0.9492495059967041, -0.8006730079650879, -0.39609119296073914, -0.6078916192054749, -0.8934920430183411, 0.6199566721916199, 1.084983229637146, 0.5772305130958557, -0.2013855278491974, -1.83936607837677, -0.39279115200042725, 0.0047060176730155945, 1.011144995689392, -1.4768304824829102, 0.5998603105545044, 0.18859830498695374, -0.7245950698852539, 0.9539254903793335, 0.3010135293006897, -0.730389416217804, 1.2178716659545898, 1.2827672958374023, 0.3830888569355011, 0.527538001537323, 1.0644817352294922, -0.3947075307369232, 0.0169195719063282, 0.864059567451477, -1.1111537218093872, -0.5707994699478149, 0.8250027894973755, -1.0374953746795654, -0.1422681212425232, 0.9872326254844666, 0.6065430641174316, 2.1273388862609863, 0.084109827876091, 1.2356139421463013, 0.02122057043015957, 0.3529229164123535, -0.40642988681793213, -0.6111922264099121, -1.2562731504440308, -1.6879280805587769, 0.9526259899139404, -0.1274711936712265, 1.5329519510269165, -0.4370756149291992, -0.49727892875671387, 0.0981450080871582, 0.19712218642234802, 1.1641995906829834, 1.2583723068237305], [0.4222254455089569, -0.7313489317893982, -1.1817700862884521, -2.790705442428589, -1.4724516868591309, 1.1885621547698975, -0.04005822911858559, 1.0466123819351196, -0.1659586876630783, -2.318033456802368, 1.505450963973999, -0.6653311252593994, -2.7010860443115234, -2.2032413482666016, 0.7712367177009583, -0.034704580903053284, -0.8494577407836914, 0.3806450068950653, -0.9700666069984436, -1.73308265209198, -0.15836773812770844, -0.49896079301834106, -0.34098631143569946, 0.48797786235809326, 0.513572633266449, 1.0019352436065674, -0.8447735905647278, 1.1038463115692139, -0.6576799154281616, 0.14474132657051086, -1.1112867593765259, 1.2168757915496826, 0.07101322710514069, -0.24141396582126617, 0.011013721115887165, -0.6020364761352539, -0.8101159334182739, -0.30549415946006775, 1.008485198020935, -0.798118531703949, 0.04141007736325264, 1.8646843433380127, -1.7040636539459229, 0.1868850290775299, 0.12198709696531296, -0.5776477456092834, 1.1411840915679932, -1.1031168699264526, -1.687846064567566, -0.796815812587738, -0.052970364689826965, -1.1567211151123047, -0.29214704036712646, 2.2259411811828613, -0.2251804769039154, 2.604926109313965, 2.886641502380371, -0.8913339972496033, -0.1405477225780487, -1.9199520349502563, 0.7076051831245422, 0.26800140738487244, 0.7495595812797546, 0.11085251718759537, -0.46535882353782654, -0.5696507096290588, -0.2923934757709503, -0.7547443509101868, 0.12592460215091705, 0.3058832287788391, -0.22127950191497803, -0.7751023769378662, -0.4888516962528229, 0.3308160603046417, 0.0031138285994529724, 0.22596202790737152, -0.26501330733299255, -0.6650277972221375, 0.5477136969566345, -1.1732407808303833, -0.1412285417318344, -0.46323469281196594, -0.12327373772859573, 0.037546806037425995, 1.8529119491577148, -0.383835107088089, -0.4007154405117035, 1.2329442501068115, 0.9493907690048218, 0.8518105745315552, 0.911828875541687, 1.324806571006775, -0.4603813588619232, -1.0092346668243408, 0.412416011095047, -1.082698941230774, -0.8071816563606262, 0.8550775647163391, 1.9876657724380493, 0.21921677887439728, -0.06524468958377838, -1.913764238357544, -0.43762490153312683, 0.2048855721950531, -0.25441640615463257, 0.4446212947368622, 1.0880972146987915, -0.7955989837646484, 0.3393387794494629, 0.12000448256731033, 0.18608945608139038, -0.4580870270729065, -1.1008341312408447, 1.2943861484527588, 0.037757668644189835, -0.061820998787879944, 0.6351805329322815, 0.1722838133573532, -0.3982310891151428, 1.5650250911712646, -1.5031548738479614, -0.4246228039264679, 0.20873118937015533, 2.222907066345215, -0.2845737636089325, 0.27180349826812744, 0.98272305727005, -1.2921173572540283]]]], "dtype": "float32"}, {"__nd__": [32, 11, 8, 19], "dtype": "int64"}]}}, {"a": [{"__nd__": [[[[1.015409231185913], [-0.9465458989143372], [-0.7648751735687256]]]], "dtype": "float32"}, {"__nd__": [[[[0.43187958002090454], [-0.438129723072052], [-0.532039225101471]]]], "dtype": "float32"}, {"__nd__": [[[[-1.2716318368911743], [1.2050848007202148], [1.5000286102294922]]]], "dtype": "float32"}, {"__nd__": [[[[-0.19267000257968903], [-0.3386070132255554], [-0.43949148058891296]]]], "dtype": "float32"}, {"__nd__": [0], "dtype": "int64"}], "e": {"__tup__": [{"__nd__": [[[[-1.2716318368911743], [1.2050848007202148], [1.5000286102294922]]]], "dtype": "float32"}, {"__nd__": [[[[-0.19267000257968903], [-0.3386070132255554], [-0.43949148058891296]]]], "dtype": "float32"}, {"__nd__": [3], "dtype": "int64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(kv_append, json.loads(_TESTS), 'triton', *[1e-06, 1e-07])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")